In [ ]:
from google.colab import drive
drive.mount('/mnt/storage')


In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
import platform
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import pandas as pd

PROJECT_ROOT = Path('/path/to/project')
RAW_DATA_ROOT = PROJECT_ROOT / 'ISLES24 dataset'
TRAIN_ROOT = RAW_DATA_ROOT / 'train'

RAW_IMAGE_ROOT = TRAIN_ROOT / 'raw_data'
DERIVATIVES_ROOT = TRAIN_ROOT / 'derivatives'
PHENOTYPE_ROOT = TRAIN_ROOT / 'phenotype'

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
    'configs': PROJECT_ROOT / 'configs',
}

print(f'PROJECT_ROOT:  {PROJECT_ROOT}')
print(f'RAW_DATA_ROOT: {RAW_DATA_ROOT}')
print(f'TRAIN_ROOT:    {TRAIN_ROOT}')


In [ ]:
def resolved(path: Path) -> Path:
    return path.expanduser().resolve(strict=False)

def is_inside(child: Path, parent: Path) -> bool:
    try:
        resolved(child).relative_to(resolved(parent))
        return True
    except ValueError:
        return False

def assert_generated_path_is_safe(path: Path) -> None:
    if not is_inside(path, PROJECT_ROOT):
        raise ValueError(f'Path is outside PROJECT_ROOT: {path}')
    if is_inside(path, RAW_DATA_ROOT):
        raise ValueError(f'Generated path is inside RAW_DATA_ROOT, forbidden: {path}')

def path_or_none(path: Path):
    return str(path) if path.exists() else None

def print_check(name: str, ok: bool, detail: str = ''):
    status = 'OK' if ok else 'FAIL'
    suffix = f' - {detail}' if detail else ''
    print(f'[{status}] {name}{suffix}')

required_roots = {
    'PROJECT_ROOT': PROJECT_ROOT,
    'RAW_DATA_ROOT': RAW_DATA_ROOT,
    'TRAIN_ROOT': TRAIN_ROOT,
    'RAW_IMAGE_ROOT': RAW_IMAGE_ROOT,
    'DERIVATIVES_ROOT': DERIVATIVES_ROOT,
    'PHENOTYPE_ROOT': PHENOTYPE_ROOT,
}

failed = []

for name, path in required_roots.items():
    ok = path.exists() and path.is_dir()
    print_check(f'{name} exists', ok, str(path))
    if not ok:
        failed.append(name)

print_check('RAW_DATA_ROOT inside PROJECT_ROOT', is_inside(RAW_DATA_ROOT, PROJECT_ROOT))

for name, path in GENERATED_DIRS.items():
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)
    print_check(f'Generated dir ready: {name}', path.exists(), str(path))

if failed:
    raise RuntimeError(f'Missing required folders: {failed}')


In [ ]:
subject_ids = sorted([
    p.name for p in DERIVATIVES_ROOT.iterdir()
    if p.is_dir() and p.name.startswith('sub-')
])

print(f'Found {len(subject_ids)} subjects in derivatives.')

rows = []

for sid in subject_ids:
    raw_ses01 = RAW_IMAGE_ROOT / sid / 'ses-01'
    raw_perf = raw_ses01 / 'perfusion-maps'

    deriv_ses01 = DERIVATIVES_ROOT / sid / 'ses-01'
    deriv_perf = deriv_ses01 / 'perfusion-maps'
    deriv_ses02 = DERIVATIVES_ROOT / sid / 'ses-02'

    phenotype_ses01 = PHENOTYPE_ROOT / sid / 'ses-01'
    phenotype_ses02 = PHENOTYPE_ROOT / sid / 'ses-02'

    rows.append({
        'subject_id': sid,
        'split': 'train',

        'path_raw_ncct': path_or_none(raw_ses01 / f'{sid}_ses-01_ncct.nii.gz'),
        'path_raw_cta': path_or_none(raw_ses01 / f'{sid}_ses-01_cta.nii.gz'),
        'path_raw_ctp': path_or_none(raw_ses01 / f'{sid}_ses-01_ctp.nii.gz'),
        'path_raw_cbf': path_or_none(raw_perf / f'{sid}_ses-01_cbf.nii.gz'),
        'path_raw_cbv': path_or_none(raw_perf / f'{sid}_ses-01_cbv.nii.gz'),
        'path_raw_mtt': path_or_none(raw_perf / f'{sid}_ses-01_mtt.nii.gz'),
        'path_raw_tmax': path_or_none(raw_perf / f'{sid}_ses-01_tmax.nii.gz'),

        'path_deriv_cta': path_or_none(deriv_ses01 / f'{sid}_ses-01_space-ncct_cta.nii.gz'),
        'path_deriv_ctp': path_or_none(deriv_ses01 / f'{sid}_ses-01_space-ncct_ctp.nii.gz'),
        'path_deriv_cbf': path_or_none(deriv_perf / f'{sid}_ses-01_space-ncct_cbf.nii.gz'),
        'path_deriv_cbv': path_or_none(deriv_perf / f'{sid}_ses-01_space-ncct_cbv.nii.gz'),
        'path_deriv_mtt': path_or_none(deriv_perf / f'{sid}_ses-01_space-ncct_mtt.nii.gz'),
        'path_deriv_tmax': path_or_none(deriv_perf / f'{sid}_ses-01_space-ncct_tmax.nii.gz'),
        'path_cow_mask': path_or_none(deriv_ses01 / f'{sid}_ses-01_space-ncct_cow-msk.nii.gz'),
        'path_lvo_mask': path_or_none(deriv_ses01 / f'{sid}_ses-01_space-ncct_lvo-msk.nii.gz'),

        'path_followup_adc': path_or_none(deriv_ses02 / f'{sid}_ses-02_space-ncct_adc.nii.gz'),
        'path_followup_dwi': path_or_none(deriv_ses02 / f'{sid}_ses-02_space-ncct_dwi.nii.gz'),
        'path_final_infarct_mask': path_or_none(deriv_ses02 / f'{sid}_ses-02_space-ncct_lesion-msk.nii.gz'),

        'path_demographic_baseline_xlsx': path_or_none(
            phenotype_ses01 / f'{sid}_ses-01_demographic_baseline.xlsx'
        ),
        'path_outcome_xlsx': path_or_none(
            phenotype_ses02 / f'{sid}_ses-02_outcome.xlsx'
        ),
    })

manifest_df = pd.DataFrame(rows)

display(manifest_df.head())
print(f'Manifest rows: {len(manifest_df)}')


In [ ]:
missing_summary = manifest_df.isna().sum().sort_values(ascending=False)
display(missing_summary.to_frame('missing_count'))

manifest_csv_path = GENERATED_DIRS['metadata'] / 'subject_manifest_paths_only_trial0.csv'
manifest_json_path = GENERATED_DIRS['metadata'] / 'subject_manifest_paths_only_trial0.json'

assert_generated_path_is_safe(manifest_csv_path)
assert_generated_path_is_safe(manifest_json_path)

manifest_df.to_csv(manifest_csv_path, index=False)
manifest_df.to_json(manifest_json_path, orient='records', indent=2)

print(f'Saved CSV:  {manifest_csv_path}')
print(f'Saved JSON: {manifest_json_path}')

if missing_summary.max() == 0:
    print('All expected path columns are present for all subjects.')
else:
    print('Some expected files are missing. Review the table above before continuing.')


In [ ]:
try:
    import nibabel as nib
except ImportError:
    !pip -q install nibabel
    import nibabel as nib

import numpy as np
import pandas as pd
from pathlib import Path

IMAGE_PATH_COLUMNS = [
    'path_raw_ncct',
    'path_raw_cta',
    'path_raw_ctp',
    'path_raw_cbf',
    'path_raw_cbv',
    'path_raw_mtt',
    'path_raw_tmax',
    'path_deriv_cta',
    'path_deriv_ctp',
    'path_deriv_cbf',
    'path_deriv_cbv',
    'path_deriv_mtt',
    'path_deriv_tmax',
    'path_cow_mask',
    'path_lvo_mask',
    'path_followup_adc',
    'path_followup_dwi',
    'path_final_infarct_mask',
]

def summarize_nifti_header(path_value):
    path = Path(path_value)

    if not path.exists():
        return {
            'exists': False,
            'readable': False,
            'shape': None,
            'spacing': None,
            'orientation': None,
            'dtype': None,
            'error': 'file does not exist',
        }

    try:
        img = nib.load(str(path))
        shape = tuple(int(x) for x in img.shape)
        spacing = tuple(float(x) for x in img.header.get_zooms()[:len(shape)])
        orientation = ''.join(nib.aff2axcodes(img.affine))
        dtype = str(img.get_data_dtype())

        return {
            'exists': True,
            'readable': True,
            'shape': shape,
            'spacing': spacing,
            'orientation': orientation,
            'dtype': dtype,
            'error': None,
        }
    except Exception as exc:
        return {
            'exists': True,
            'readable': False,
            'shape': None,
            'spacing': None,
            'orientation': None,
            'dtype': None,
            'error': repr(exc),
        }

def summarize_mask_values(path_value):
    path = Path(path_value)

    try:
        img = nib.load(str(path))
        data = np.asanyarray(img.dataobj)

        finite = np.isfinite(data)
        unique_values = np.unique(data[finite])

        nonzero_voxels = int(np.count_nonzero(data))
        voxel_volume_mm3 = float(np.prod(img.header.get_zooms()[:3]))
        nonzero_volume_ml = nonzero_voxels * voxel_volume_mm3 / 1000.0

        return {
            'mask_unique_values': unique_values[:20].tolist(),
            'mask_unique_value_count': int(len(unique_values)),
            'mask_nonzero_voxels': nonzero_voxels,
            'mask_nonzero_volume_ml': nonzero_volume_ml,
            'mask_error': None,
        }
    except Exception as exc:
        return {
            'mask_unique_values': None,
            'mask_unique_value_count': None,
            'mask_nonzero_voxels': None,
            'mask_nonzero_volume_ml': None,
            'mask_error': repr(exc),
        }

validation_rows = []

for idx, (_, row) in enumerate(manifest_df.iterrows(), start=1):
    sid = row['subject_id']

    if idx == 1 or idx % 10 == 0 or idx == len(manifest_df):
        print(f'Validating headers {idx}/{len(manifest_df)}: {sid}')

    for col in IMAGE_PATH_COLUMNS:
        validation_rows.append({
            'subject_id': sid,
            'path_column': col,
            'path': row[col],
            **summarize_nifti_header(row[col]),
        })

validation_df = pd.DataFrame(validation_rows)

target_rows = []

for idx, (_, row) in enumerate(manifest_df.iterrows(), start=1):
    sid = row['subject_id']

    if idx == 1 or idx % 10 == 0 or idx == len(manifest_df):
        print(f'Validating target mask {idx}/{len(manifest_df)}: {sid}')

    target_rows.append({
        'subject_id': sid,
        'path_final_infarct_mask': row['path_final_infarct_mask'],
        **summarize_mask_values(row['path_final_infarct_mask']),
    })

target_mask_df = pd.DataFrame(target_rows)

validation_csv_path = GENERATED_DIRS['metadata'] / 'nifti_header_validation_trial0.csv'
target_csv_path = GENERATED_DIRS['metadata'] / 'target_mask_validation_trial0.csv'

assert_generated_path_is_safe(validation_csv_path)
assert_generated_path_is_safe(target_csv_path)

validation_df.to_csv(validation_csv_path, index=False)
target_mask_df.to_csv(target_csv_path, index=False)

print(f'Validation rows: {len(validation_df)}')
print(f'Saved NIfTI header validation: {validation_csv_path}')
print(f'Saved target mask validation: {target_csv_path}')

print('\nUnreadable files by path column:')
display(
    validation_df[~validation_df['readable']]
    .groupby('path_column')
    .size()
    .to_frame('unreadable_count')
)

print('\nOrientation counts by path column:')
display(
    validation_df
    .groupby(['path_column', 'orientation'])
    .size()
    .reset_index(name='count')
    .sort_values(['path_column', 'count'], ascending=[True, False])
)

print('\nTarget mask lesion volume summary, ml:')
display(target_mask_df['mask_nonzero_volume_ml'].describe())

print('\nSmallest target masks:')
display(target_mask_df.sort_values('mask_nonzero_volume_ml').head(10))

print('\nLargest target masks:')
display(target_mask_df.sort_values('mask_nonzero_volume_ml', ascending=False).head(10))


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import nibabel as nib

SPATIAL_ALIGNMENT_COLUMNS = [
    'path_raw_ncct',
    'path_deriv_cta',
    'path_deriv_ctp',
    'path_deriv_cbf',
    'path_deriv_cbv',
    'path_deriv_mtt',
    'path_deriv_tmax',
    'path_cow_mask',
    'path_lvo_mask',
    'path_final_infarct_mask',
]

def load_spatial_geometry(path_value):
    path = Path(path_value)
    img = nib.load(str(path))

    full_shape = tuple(int(x) for x in img.shape)
    spatial_shape = full_shape[:3]
    spacing = tuple(float(x) for x in img.header.get_zooms()[:3])
    orientation = ''.join(nib.aff2axcodes(img.affine))
    affine = img.affine
    timepoints = full_shape[3] if len(full_shape) == 4 else None

    return {
        'full_shape': full_shape,
        'spatial_shape': spatial_shape,
        'spacing': spacing,
        'orientation': orientation,
        'affine': affine,
        'timepoints': timepoints,
    }

spatial_alignment_rows = []

for _, row in manifest_df.iterrows():
    sid = row['subject_id']

    ref_col = 'path_raw_ncct'
    ref = load_spatial_geometry(row[ref_col])

    for col in SPATIAL_ALIGNMENT_COLUMNS:
        geom = load_spatial_geometry(row[col])

        same_spatial_shape = geom['spatial_shape'] == ref['spatial_shape']
        same_spacing = np.allclose(geom['spacing'], ref['spacing'], atol=1e-4)
        same_orientation = geom['orientation'] == ref['orientation']
        same_affine = np.allclose(geom['affine'], ref['affine'], atol=1e-4)

        spatial_alignment_rows.append({
            'subject_id': sid,
            'reference_column': ref_col,
            'path_column': col,
            'full_shape': geom['full_shape'],
            'spatial_shape': geom['spatial_shape'],
            'spacing': geom['spacing'],
            'orientation': geom['orientation'],
            'timepoints': geom['timepoints'],
            'same_spatial_shape_as_raw_ncct': same_spatial_shape,
            'same_spacing_as_raw_ncct': same_spacing,
            'same_orientation_as_raw_ncct': same_orientation,
            'same_affine_as_raw_ncct': same_affine,
            'spatially_aligned_to_raw_ncct': (
                same_spatial_shape and same_spacing and same_orientation and same_affine
            ),
        })

spatial_alignment_df = pd.DataFrame(spatial_alignment_rows)

spatial_alignment_csv_path = GENERATED_DIRS['metadata'] / 'within_subject_spatial_alignment_trial0.csv'
assert_generated_path_is_safe(spatial_alignment_csv_path)
spatial_alignment_df.to_csv(spatial_alignment_csv_path, index=False)

print(f'Saved corrected spatial alignment validation: {spatial_alignment_csv_path}')

print('\nCorrected spatial alignment summary by path column:')
display(
    spatial_alignment_df
    .groupby('path_column')[
        [
            'same_spatial_shape_as_raw_ncct',
            'same_spacing_as_raw_ncct',
            'same_orientation_as_raw_ncct',
            'same_affine_as_raw_ncct',
            'spatially_aligned_to_raw_ncct',
        ]
    ]
    .sum()
    .astype(int)
)

print('\nCTP timepoint count distribution:')
display(
    spatial_alignment_df[spatial_alignment_df['path_column'] == 'path_deriv_ctp']
    .groupby('timepoints')
    .size()
    .reset_index(name='subject_count')
    .sort_values('timepoints')
)

print('\nFiles not spatially aligned to raw NCCT:')
not_spatially_aligned_df = spatial_alignment_df[
    ~spatial_alignment_df['spatially_aligned_to_raw_ncct']
]
display(not_spatially_aligned_df)

print('\nSubjects with any corrected spatial alignment issue:')
problem_subjects = (
    not_spatially_aligned_df['subject_id']
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print(problem_subjects)
print(f'Count: {len(problem_subjects)}')


In [ ]:
import pandas as pd
from pathlib import Path

if 'target_mask_df' not in globals():
    target_mask_df = pd.read_csv(GENERATED_DIRS['metadata'] / 'target_mask_validation_trial0.csv')

target_mask_df = target_mask_df.copy()
target_mask_df['mask_nonzero_volume_ml'] = pd.to_numeric(
    target_mask_df['mask_nonzero_volume_ml'],
    errors='coerce'
)

empty_subjects = (
    target_mask_df[target_mask_df['mask_nonzero_volume_ml'] == 0]
    .sort_values('subject_id')
    ['subject_id']
    .tolist()
)

small_nonempty_subjects = (
    target_mask_df[target_mask_df['mask_nonzero_volume_ml'] > 0]
    .sort_values('mask_nonzero_volume_ml')
    .head(3)
    ['subject_id']
    .tolist()
)

median_volume = target_mask_df['mask_nonzero_volume_ml'].median()
median_subject = (
    target_mask_df.assign(
        distance_to_median=(target_mask_df['mask_nonzero_volume_ml'] - median_volume).abs()
    )
    .sort_values('distance_to_median')
    .iloc[0]['subject_id']
)

large_subjects = (
    target_mask_df
    .sort_values('mask_nonzero_volume_ml', ascending=False)
    .head(3)
    ['subject_id']
    .tolist()
)

special_subjects = ['sub-stroke0016']

visual_subjects = []

for sid in empty_subjects + small_nonempty_subjects + [median_subject] + large_subjects + special_subjects:
    if sid not in visual_subjects:
        visual_subjects.append(sid)

print('Selected subjects for visual QA:')
for sid in visual_subjects:
    vol = target_mask_df.loc[
        target_mask_df['subject_id'] == sid,
        'mask_nonzero_volume_ml'
    ].iloc[0]
    print(f'  {sid}: {vol:.4f} mL')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from pathlib import Path

VISUAL_QA_DIR = GENERATED_DIRS['outputs'] / 'visual_qa_trial0'
assert_generated_path_is_safe(VISUAL_QA_DIR)
VISUAL_QA_DIR.mkdir(parents=True, exist_ok=True)

VISUAL_MODALITIES = [
    ('NCCT', 'path_raw_ncct'),
    ('CTA', 'path_deriv_cta'),
    ('CBF', 'path_deriv_cbf'),
    ('CBV', 'path_deriv_cbv'),
    ('MTT', 'path_deriv_mtt'),
    ('Tmax', 'path_deriv_tmax'),
    ('Target', 'path_final_infarct_mask'),
]

def load_volume(path_value):
    img = nib.load(str(path_value))
    data = np.asanyarray(img.dataobj)

    if data.ndim == 4:
        data = data[..., data.shape[-1] // 2]

    return data

def normalize_for_display(x):
    x = np.asarray(x, dtype=np.float32)

    finite = np.isfinite(x)
    if not finite.any():
        return np.zeros_like(x, dtype=np.float32)

    vals = x[finite]
    lo, hi = np.percentile(vals, [1, 99])

    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)

    x = np.clip(x, lo, hi)
    return (x - lo) / (hi - lo)

def best_lesion_slice(mask):
    if np.count_nonzero(mask) == 0:
        return mask.shape[2] // 2

    counts = mask.sum(axis=(0, 1))
    return int(np.argmax(counts))

for sid in visual_subjects:
    row = manifest_df[manifest_df['subject_id'] == sid].iloc[0]

    target = load_volume(row['path_final_infarct_mask'])
    z = best_lesion_slice(target)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()

    for ax, (label, col) in zip(axes, VISUAL_MODALITIES):
        vol = load_volume(row[col])
        slc = vol[:, :, min(z, vol.shape[2] - 1)]

        if label == 'Target':
            ax.imshow(slc.T, cmap='gray', origin='lower')
        else:
            ax.imshow(normalize_for_display(slc).T, cmap='gray', origin='lower')

        ax.set_title(f'{label} z={z}')
        ax.axis('off')

    ncct = load_volume(row['path_raw_ncct'])
    ncct_slice = ncct[:, :, min(z, ncct.shape[2] - 1)]
    target_slice = target[:, :, min(z, target.shape[2] - 1)]

    axes[-1].imshow(normalize_for_display(ncct_slice).T, cmap='gray', origin='lower')
    axes[-1].imshow(
        np.ma.masked_where(target_slice.T <= 0, target_slice.T),
        cmap='autumn',
        alpha=0.55,
        origin='lower'
    )
    axes[-1].set_title('NCCT + target overlay')
    axes[-1].axis('off')

    fig.suptitle(f'{sid} visual QA', fontsize=16)
    fig.tight_layout()

    out_path = VISUAL_QA_DIR / f'{sid}_visual_qa.png'
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    print(f'Saved: {out_path}')


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

SPLIT_SEED = 20240824
VAL_FRACTION = 0.20
SPLIT_NAME = f'trial0_seed{SPLIT_SEED}_val{int(VAL_FRACTION * 100)}'

split_dir = GENERATED_DIRS['splits'] / SPLIT_NAME
assert_generated_path_is_safe(split_dir)
split_dir.mkdir(parents=True, exist_ok=True)

if 'target_mask_df' not in globals():
    target_mask_df = pd.read_csv(GENERATED_DIRS['metadata'] / 'target_mask_validation_trial0.csv')

split_df = manifest_df.merge(
    target_mask_df[['subject_id', 'mask_nonzero_volume_ml', 'mask_nonzero_voxels']],
    on='subject_id',
    how='left'
)

def lesion_volume_bin(volume_ml):
    if volume_ml == 0:
        return 'empty'
    if volume_ml < 1:
        return 'tiny'
    if volume_ml < 5:
        return 'small'
    if volume_ml < 25:
        return 'medium'
    if volume_ml < 75:
        return 'large'
    return 'very_large'

split_df['lesion_volume_bin'] = split_df['mask_nonzero_volume_ml'].apply(lesion_volume_bin)
split_df['official_split'] = 'train'
split_df['dev_split'] = 'train'

rng = np.random.default_rng(SPLIT_SEED)

val_ids = []

for bin_name, group in split_df.groupby('lesion_volume_bin'):
    ids = group['subject_id'].to_numpy()
    rng.shuffle(ids)

    if len(ids) <= 2:
        n_val = 1 if len(ids) > 1 else 0
    else:
        n_val = max(1, int(round(len(ids) * VAL_FRACTION)))

    val_ids.extend(ids[:n_val].tolist())

split_df.loc[split_df['subject_id'].isin(val_ids), 'dev_split'] = 'val'

train_ids = split_df.loc[split_df['dev_split'] == 'train', 'subject_id'].sort_values().tolist()
val_ids = split_df.loc[split_df['dev_split'] == 'val', 'subject_id'].sort_values().tolist()

train_ids_path = split_dir / 'train_ids.csv'
val_ids_path = split_dir / 'val_ids.csv'
split_manifest_path = split_dir / 'split_manifest.csv'
split_config_path = split_dir / 'split_config.json'

for path in [train_ids_path, val_ids_path, split_manifest_path, split_config_path]:
    assert_generated_path_is_safe(path)

pd.DataFrame({'subject_id': train_ids}).to_csv(train_ids_path, index=False)
pd.DataFrame({'subject_id': val_ids}).to_csv(val_ids_path, index=False)
split_df.to_csv(split_manifest_path, index=False)

split_config = {
    'split_name': SPLIT_NAME,
    'split_seed': SPLIT_SEED,
    'val_fraction': VAL_FRACTION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'official_split': 'train',
    'dev_split_column': 'dev_split',
    'stratification_column': 'lesion_volume_bin',
    'notes': 'Subject-level train/val split. No held-out test split created from public training data.',
}

split_config_path.write_text(json.dumps(split_config, indent=2), encoding='utf-8')

print(f'Split name: {SPLIT_NAME}')
print(f'Train subjects: {len(train_ids)}')
print(f'Val subjects:   {len(val_ids)}')

print('\nSplit balance by lesion volume bin:')
display(
    split_df
    .groupby(['lesion_volume_bin', 'dev_split'])
    .size()
    .reset_index(name='count')
    .sort_values(['lesion_volume_bin', 'dev_split'])
)

print('\nSaved:')
print(train_ids_path)
print(val_ids_path)
print(split_manifest_path)
print(split_config_path)


In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path

SPLIT_NAME = 'trial0_seed20240824_val20'
split_manifest_path = GENERATED_DIRS['splits'] / SPLIT_NAME / 'split_manifest.csv'

split_df = pd.read_csv(split_manifest_path)

BASELINE_INPUT_COLUMNS = [
    'path_raw_ncct',
    'path_deriv_cta',
    'path_deriv_cbf',
    'path_deriv_cbv',
    'path_deriv_mtt',
    'path_deriv_tmax',
]

PROFILE_SEED = 20240824
PROFILE_SUBJECT_COUNT = 30

train_subjects = (
    split_df[split_df['dev_split'] == 'train']
    ['subject_id']
    .sort_values()
    .tolist()
)

rng = np.random.default_rng(PROFILE_SEED)
profile_subjects = sorted(
    rng.choice(
        train_subjects,
        size=min(PROFILE_SUBJECT_COUNT, len(train_subjects)),
        replace=False
    ).tolist()
)

print(f'Profiling {len(profile_subjects)} training subjects.')
print(profile_subjects)

profile_rows = []

def load_data(path_value):
    img = nib.load(str(path_value))
    data = np.asanyarray(img.dataobj)

    if data.ndim == 4:
        data = data[..., data.shape[-1] // 2]

    return data.astype(np.float32)

for idx, sid in enumerate(profile_subjects, start=1):
    row = manifest_df[manifest_df['subject_id'] == sid].iloc[0]

    print(f'Profiling subject {idx}/{len(profile_subjects)}: {sid}')

    for col in BASELINE_INPUT_COLUMNS:
        data = load_data(row[col])
        finite = np.isfinite(data)

        if not finite.any():
            profile_rows.append({
                'subject_id': sid,
                'path_column': col,
                'finite_voxels': 0,
                'min': None,
                'p0_5': None,
                'p1': None,
                'p5': None,
                'p50': None,
                'p95': None,
                'p99': None,
                'p99_5': None,
                'max': None,
                'mean': None,
                'std': None,
            })
            continue

        vals = data[finite]

        profile_rows.append({
            'subject_id': sid,
            'path_column': col,
            'finite_voxels': int(vals.size),
            'min': float(np.min(vals)),
            'p0_5': float(np.percentile(vals, 0.5)),
            'p1': float(np.percentile(vals, 1)),
            'p5': float(np.percentile(vals, 5)),
            'p50': float(np.percentile(vals, 50)),
            'p95': float(np.percentile(vals, 95)),
            'p99': float(np.percentile(vals, 99)),
            'p99_5': float(np.percentile(vals, 99.5)),
            'max': float(np.max(vals)),
            'mean': float(np.mean(vals)),
            'std': float(np.std(vals)),
        })

intensity_profile_df = pd.DataFrame(profile_rows)

intensity_profile_path = GENERATED_DIRS['metadata'] / 'intensity_profile_trial0.csv'
assert_generated_path_is_safe(intensity_profile_path)
intensity_profile_df.to_csv(intensity_profile_path, index=False)

print(f'Saved intensity profile: {intensity_profile_path}')

print('\nIntensity summary by channel:')
summary_cols = ['min', 'p0_5', 'p1', 'p5', 'p50', 'p95', 'p99', 'p99_5', 'max', 'mean', 'std']

display(
    intensity_profile_df
    .groupby('path_column')[summary_cols]
    .median()
    .reset_index()
)

print('\nExtreme values by channel:')
display(
    intensity_profile_df
    .groupby('path_column')
    .agg(
        global_min=('min', 'min'),
        median_p1=('p1', 'median'),
        median_p99=('p99', 'median'),
        global_max=('max', 'max'),
    )
    .reset_index()
)


In [ ]:
import json
from datetime import datetime, timezone

PREPROCESSING_VERSION = 'v1_trial0'

PREPROCESSING_CONFIG = {
    'preprocessing_version': PREPROCESSING_VERSION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),

    'project_root': str(PROJECT_ROOT),
    'raw_data_root': str(RAW_DATA_ROOT),

    'input_channels': [
        'path_raw_ncct',
        'path_deriv_cta',
        'path_deriv_cbf',
        'path_deriv_cbv',
        'path_deriv_mtt',
        'path_deriv_tmax',
    ],

    'excluded_from_trial0_inputs': {
        'path_deriv_ctp': '4D time-series CTP; defer until later ablation.',
        'path_raw_ctp': 'raw 4D CTP; defer until later ablation.',
        'path_raw_cta': 'derivative CTA is already in NCCT space.',
        'path_raw_cbf': 'derivative CBF is already in NCCT space.',
        'path_raw_cbv': 'derivative CBV is already in NCCT space.',
        'path_raw_mtt': 'derivative MTT is already in NCCT space.',
        'path_raw_tmax': 'derivative Tmax is already in NCCT space.',
        'path_cow_mask': 'one subject has COW mask spatial shape mismatch; defer.',
        'path_lvo_mask': 'use later as vascular/occlusion ablation, not first baseline.',
        'path_followup_adc': 'follow-up imaging; not valid acute input.',
        'path_followup_dwi': 'follow-up imaging; not valid acute input.',
    },

    'target_column': 'path_final_infarct_mask',

    'clip_ranges': {
        'path_raw_ncct': [-100, 300],
        'path_deriv_cta': [-100, 1000],
        'path_deriv_cbf': [0, 100],
        'path_deriv_cbv': [0, 20],
        'path_deriv_mtt': [0, 25],
        'path_deriv_tmax': [0, 20],
    },

    'normalization': {
        'type': 'clip_then_minmax_to_0_1',
        'note': 'Values are clipped per channel, then scaled to [0, 1].'
    },

    'spatial_plan': {
        'orientation': 'RAS',
        'target_spacing': [1.0, 1.0, 2.0],
        'training_patch_size': [128, 128, 64],
        'foreground_crop_source': 'path_raw_ncct',
        'foreground_threshold': -500,
        'foreground_margin_voxels_native': [16, 16, 8],
        'note': 'Use raw NCCT non-air foreground for crop; do not write into raw dataset.'
    },

    'split_name': SPLIT_NAME,
    'known_flags': {
        'empty_target_masks': ['sub-stroke0141', 'sub-stroke0162'],
        'near_empty_target_masks': ['sub-stroke0171'],
        'cow_mask_shape_mismatch': ['sub-stroke0016'],
    }
}

config_path = GENERATED_DIRS['configs'] / f'preprocessing_{PREPROCESSING_VERSION}.json'
assert_generated_path_is_safe(config_path)

config_path.write_text(json.dumps(PREPROCESSING_CONFIG, indent=2), encoding='utf-8')

print(f'Saved preprocessing config: {config_path}')
print(json.dumps(PREPROCESSING_CONFIG, indent=2))


In [ ]:
try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    ConcatItemsd,
    EnsureTyped,
)

PREPROCESSING_VERSION = 'v1_trial0'
preproc_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION
test_preproc_dir = preproc_root / 'test_subjects'

assert_generated_path_is_safe(preproc_root)
assert_generated_path_is_safe(test_preproc_dir)

test_preproc_dir.mkdir(parents=True, exist_ok=True)

config_path = GENERATED_DIRS['configs'] / f'preprocessing_{PREPROCESSING_VERSION}.json'
config = json.loads(config_path.read_text(encoding='utf-8'))

input_columns = config['input_channels']
target_column = config['target_column']
clip_ranges = config['clip_ranges']
target_spacing = tuple(config['spatial_plan']['target_spacing'])


TEST_PREPROCESS_SUBJECTS = [
    'sub-stroke0141',  
    'sub-stroke0171',  
    'sub-stroke0014',  
    'sub-stroke0117',  
]


TEST_PREPROCESS_SUBJECTS = [
    sid for sid in TEST_PREPROCESS_SUBJECTS
    if sid in set(manifest_df['subject_id'])
]

print('Testing preprocessing on subjects:')
print(TEST_PREPROCESS_SUBJECTS)

def make_case_dict(row):
    case = {
        'subject_id': row['subject_id'],
        'target': row[target_column],
    }

    for col in input_columns:
        case[col] = row[col]

    return case

def build_preprocess_transform():
    keys = input_columns + ['target']

    transforms = [
        LoadImaged(keys=keys, image_only=False),
        EnsureChannelFirstd(keys=keys),
        Orientationd(keys=keys, axcodes='RAS'),
        Spacingd(
            keys=keys,
            pixdim=target_spacing,
            mode=['bilinear'] * len(input_columns) + ['nearest'],
        ),
    ]

    for col in input_columns:
        lo, hi = clip_ranges[col]
        transforms.append(
            ScaleIntensityRanged(
                keys=[col],
                a_min=lo,
                a_max=hi,
                b_min=0.0,
                b_max=1.0,
                clip=True,
            )
        )

    transforms.extend([
        ConcatItemsd(keys=input_columns, name='image', dim=0),
        EnsureTyped(keys=['image', 'target'], dtype=torch.float32),
    ])

    return Compose(transforms)

preprocess_transform = build_preprocess_transform()

test_rows = []

for sid in TEST_PREPROCESS_SUBJECTS:
    row = manifest_df[manifest_df['subject_id'] == sid].iloc[0]
    case = make_case_dict(row)

    print(f'\nPreprocessing {sid}...')
    out = preprocess_transform(case)

    image = out['image'].cpu().numpy()
    target = out['target'].cpu().numpy()

    
    target = (target > 0.5).astype(np.uint8)

    out_path = test_preproc_dir / f'{sid}_preprocessed_v1_trial0.npz'
    assert_generated_path_is_safe(out_path)

    np.savez_compressed(
        out_path,
        image=image.astype(np.float32),
        target=target,
        subject_id=sid,
        input_columns=np.array(input_columns),
    )

    test_rows.append({
        'subject_id': sid,
        'output_path': str(out_path),
        'image_shape': tuple(image.shape),
        'target_shape': tuple(target.shape),
        'image_min': float(image.min()),
        'image_max': float(image.max()),
        'image_mean': float(image.mean()),
        'target_unique': np.unique(target).tolist(),
        'target_nonzero_voxels': int(np.count_nonzero(target)),
    })

    print(f'  image shape:  {image.shape}')
    print(f'  target shape: {target.shape}')
    print(f'  image range:  {image.min():.4f} to {image.max():.4f}')
    print(f'  target voxels: {np.count_nonzero(target)}')
    print(f'  saved: {out_path}')

test_preproc_df = pd.DataFrame(test_rows)

test_summary_path = test_preproc_dir / 'test_preprocessing_summary.csv'
assert_generated_path_is_safe(test_summary_path)
test_preproc_df.to_csv(test_summary_path, index=False)

print('\nTest preprocessing summary:')
display(test_preproc_df)

print(f'\nSaved summary: {test_summary_path}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PREPROCESSED_CHANNEL_NAMES = ['NCCT', 'CTA', 'CBF', 'CBV', 'MTT', 'Tmax']

def best_target_slice(target):
    
    mask = target[0]

    if np.count_nonzero(mask) == 0:
        return mask.shape[2] // 2

    counts = mask.sum(axis=(0, 1))
    return int(np.argmax(counts))

for _, row in test_preproc_df.iterrows():
    sid = row['subject_id']
    arr = np.load(row['output_path'], allow_pickle=True)

    image = arr['image']
    target = arr['target']

    z = best_target_slice(target)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()

    for i, name in enumerate(PREPROCESSED_CHANNEL_NAMES):
        axes[i].imshow(image[i, :, :, z].T, cmap='gray', origin='lower', vmin=0, vmax=1)
        axes[i].set_title(f'{name} z={z}')
        axes[i].axis('off')

    axes[6].imshow(target[0, :, :, z].T, cmap='gray', origin='lower')
    axes[6].set_title('Target')
    axes[6].axis('off')

    axes[7].imshow(image[0, :, :, z].T, cmap='gray', origin='lower', vmin=0, vmax=1)
    axes[7].imshow(
        np.ma.masked_where(target[0, :, :, z].T <= 0, target[0, :, :, z].T),
        cmap='autumn',
        alpha=0.55,
        origin='lower'
    )
    axes[7].set_title('NCCT + target')
    axes[7].axis('off')

    fig.suptitle(f'{sid} preprocessed v1 Trial0', fontsize=16)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path

channel_names = ['NCCT', 'CTA', 'CBF', 'CBV', 'MTT', 'Tmax']

print('Checking saved preprocessed test outputs:')
debug_rows = []

for _, row in test_preproc_df.iterrows():
    sid = row['subject_id']
    arr = np.load(row['output_path'], allow_pickle=True)

    image = arr['image']
    target = arr['target']

    for c, name in enumerate(channel_names):
        x = image[c]
        debug_rows.append({
            'subject_id': sid,
            'channel': name,
            'nan_count': int(np.isnan(x).sum()),
            'inf_count': int(np.isinf(x).sum()),
            'finite_min': float(np.nanmin(x)) if np.isfinite(x).any() else None,
            'finite_max': float(np.nanmax(x)) if np.isfinite(x).any() else None,
            'finite_mean': float(np.nanmean(x)) if np.isfinite(x).any() else None,
        })

    debug_rows.append({
        'subject_id': sid,
        'channel': 'TARGET',
        'nan_count': int(np.isnan(target).sum()),
        'inf_count': int(np.isinf(target).sum()),
        'finite_min': float(np.nanmin(target)) if np.isfinite(target).any() else None,
        'finite_max': float(np.nanmax(target)) if np.isfinite(target).any() else None,
        'finite_mean': float(np.nanmean(target)) if np.isfinite(target).any() else None,
    })

debug_df = pd.DataFrame(debug_rows)
display(debug_df)

print('\nRows with NaN or Inf:')
display(debug_df[(debug_df['nan_count'] > 0) | (debug_df['inf_count'] > 0)])


In [ ]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    ConcatItemsd,
    EnsureTyped,
    Lambdad,
)

def finite_cleanup(x):
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

def build_preprocess_transform_clean():
    keys = input_columns + ['target']

    transforms = [
        LoadImaged(keys=keys, image_only=False),
        EnsureChannelFirstd(keys=keys),
        Lambdad(keys=input_columns, func=finite_cleanup),
        Orientationd(keys=keys, axcodes='RAS'),
        Spacingd(
            keys=keys,
            pixdim=target_spacing,
            mode=['bilinear'] * len(input_columns) + ['nearest'],
        ),
        Lambdad(keys=input_columns, func=finite_cleanup),
    ]

    for col in input_columns:
        lo, hi = clip_ranges[col]
        transforms.append(
            ScaleIntensityRanged(
                keys=[col],
                a_min=lo,
                a_max=hi,
                b_min=0.0,
                b_max=1.0,
                clip=True,
            )
        )

    transforms.extend([
        Lambdad(keys=input_columns, func=finite_cleanup),
        ConcatItemsd(keys=input_columns, name='image', dim=0),
        EnsureTyped(keys=['image', 'target'], dtype=torch.float32),
    ])

    return Compose(transforms)

preprocess_transform = build_preprocess_transform_clean()

clean_test_preproc_dir = preproc_root / 'test_subjects_clean'
assert_generated_path_is_safe(clean_test_preproc_dir)
clean_test_preproc_dir.mkdir(parents=True, exist_ok=True)

clean_rows = []

for sid in TEST_PREPROCESS_SUBJECTS:
    row = manifest_df[manifest_df['subject_id'] == sid].iloc[0]
    case = make_case_dict(row)

    original_target = np.asanyarray(nib.load(row[target_column]).dataobj)
    original_target_voxels = int(np.count_nonzero(original_target))

    print(f'\nPreprocessing clean {sid}...')
    out = preprocess_transform(case)

    image = out['image'].cpu().numpy()
    target = out['target'].cpu().numpy()
    target = (target > 0.5).astype(np.uint8)

    out_path = clean_test_preproc_dir / f'{sid}_preprocessed_v1_trial0_clean.npz'
    assert_generated_path_is_safe(out_path)

    np.savez_compressed(
        out_path,
        image=image.astype(np.float32),
        target=target,
        subject_id=sid,
        input_columns=np.array(input_columns),
    )

    clean_rows.append({
        'subject_id': sid,
        'output_path': str(out_path),
        'image_shape': tuple(image.shape),
        'target_shape': tuple(target.shape),
        'image_nan_count': int(np.isnan(image).sum()),
        'image_inf_count': int(np.isinf(image).sum()),
        'image_min': float(np.nanmin(image)),
        'image_max': float(np.nanmax(image)),
        'image_mean': float(np.nanmean(image)),
        'original_target_voxels': original_target_voxels,
        'preprocessed_target_voxels': int(np.count_nonzero(target)),
        'target_unique': np.unique(target).tolist(),
    })

clean_test_preproc_df = pd.DataFrame(clean_rows)
display(clean_test_preproc_df)


In [ ]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
    ConcatItemsd,
    EnsureTyped,
    Lambdad,
)

def finite_cleanup(x):
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

def build_preprocess_transform_clean_v2():
    keys = input_columns + ['target']

    transforms = [
        LoadImaged(keys=keys, image_only=False),
        EnsureChannelFirstd(keys=keys),
        Orientationd(keys=keys, axcodes='RAS'),
        Spacingd(
            keys=keys,
            pixdim=target_spacing,
            mode=['bilinear'] * len(input_columns) + ['nearest'],
        ),

        
        Lambdad(keys=input_columns, func=finite_cleanup),
    ]

    for col in input_columns:
        lo, hi = clip_ranges[col]
        transforms.append(
            ScaleIntensityRanged(
                keys=[col],
                a_min=lo,
                a_max=hi,
                b_min=0.0,
                b_max=1.0,
                clip=True,
            )
        )

    transforms.extend([
        Lambdad(keys=input_columns, func=finite_cleanup),
        ConcatItemsd(keys=input_columns, name='image', dim=0),
        EnsureTyped(keys=['image', 'target'], dtype=torch.float32),
    ])

    return Compose(transforms)

preprocess_transform = build_preprocess_transform_clean_v2()

clean_test_preproc_dir = preproc_root / 'test_subjects_clean_v2'
assert_generated_path_is_safe(clean_test_preproc_dir)
clean_test_preproc_dir.mkdir(parents=True, exist_ok=True)

clean_rows = []

for sid in TEST_PREPROCESS_SUBJECTS:
    row = manifest_df[manifest_df['subject_id'] == sid].iloc[0]
    case = make_case_dict(row)

    original_target = np.asanyarray(nib.load(row[target_column]).dataobj)
    original_target_voxels = int(np.count_nonzero(original_target))

    print(f'\nPreprocessing clean v2 {sid}...')
    out = preprocess_transform(case)

    image = out['image'].cpu().numpy()
    target = out['target'].cpu().numpy()
    target = (target > 0.5).astype(np.uint8)

    shape_match = image.shape[1:] == target.shape[1:]

    out_path = clean_test_preproc_dir / f'{sid}_preprocessed_v1_trial0_clean_v2.npz'
    assert_generated_path_is_safe(out_path)

    np.savez_compressed(
        out_path,
        image=image.astype(np.float32),
        target=target,
        subject_id=sid,
        input_columns=np.array(input_columns),
    )

    clean_rows.append({
        'subject_id': sid,
        'output_path': str(out_path),
        'image_shape': tuple(image.shape),
        'target_shape': tuple(target.shape),
        'image_target_spatial_shape_match': shape_match,
        'image_nan_count': int(np.isnan(image).sum()),
        'image_inf_count': int(np.isinf(image).sum()),
        'image_min': float(np.nanmin(image)),
        'image_max': float(np.nanmax(image)),
        'image_mean': float(np.nanmean(image)),
        'original_target_voxels': original_target_voxels,
        'preprocessed_target_voxels': int(np.count_nonzero(target)),
        'target_unique': np.unique(target).tolist(),
    })

clean_test_preproc_df = pd.DataFrame(clean_rows)
display(clean_test_preproc_df)

if not clean_test_preproc_df['image_target_spatial_shape_match'].all():
    raise RuntimeError('Image/target shape mismatch remains. Do not batch preprocess.')

if clean_test_preproc_df['image_nan_count'].sum() > 0:
    raise RuntimeError('NaNs remain in preprocessed image. Do not batch preprocess.')


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

SPLIT_SEED = 20240824
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
SPLIT_NAME = f'trial0_seed{SPLIT_SEED}_train70_val15_test15'

split_dir = GENERATED_DIRS['splits'] / SPLIT_NAME
assert_generated_path_is_safe(split_dir)
split_dir.mkdir(parents=True, exist_ok=True)

if 'target_mask_df' not in globals():
    target_mask_df = pd.read_csv(GENERATED_DIRS['metadata'] / 'target_mask_validation_trial0.csv')

split_df = manifest_df.merge(
    target_mask_df[['subject_id', 'mask_nonzero_volume_ml', 'mask_nonzero_voxels']],
    on='subject_id',
    how='left'
)

def lesion_volume_bin(volume_ml):
    if volume_ml == 0:
        return 'empty'
    if volume_ml < 1:
        return 'tiny'
    if volume_ml < 5:
        return 'small'
    if volume_ml < 25:
        return 'medium'
    if volume_ml < 75:
        return 'large'
    return 'very_large'

split_df['lesion_volume_bin'] = split_df['mask_nonzero_volume_ml'].apply(lesion_volume_bin)
split_df['official_split'] = 'train'
split_df['dev_split'] = 'train'

rng = np.random.default_rng(SPLIT_SEED)

val_ids = []
test_ids = []

for bin_name, group in split_df.groupby('lesion_volume_bin'):
    ids = group['subject_id'].to_numpy()
    rng.shuffle(ids)

    n = len(ids)

    if n <= 2:
        n_val = 1 if n >= 2 else 0
        n_test = 0
    else:
        n_val = max(1, int(round(n * VAL_FRACTION)))
        n_test = max(1, int(round(n * TEST_FRACTION)))

        if n_val + n_test >= n:
            n_test = max(0, n - n_val - 1)

    val_ids.extend(ids[:n_val].tolist())
    test_ids.extend(ids[n_val:n_val + n_test].tolist())

split_df.loc[split_df['subject_id'].isin(val_ids), 'dev_split'] = 'val'
split_df.loc[split_df['subject_id'].isin(test_ids), 'dev_split'] = 'test'

train_ids = split_df.loc[split_df['dev_split'] == 'train', 'subject_id'].sort_values().tolist()
val_ids = split_df.loc[split_df['dev_split'] == 'val', 'subject_id'].sort_values().tolist()
test_ids = split_df.loc[split_df['dev_split'] == 'test', 'subject_id'].sort_values().tolist()

train_ids_path = split_dir / 'train_ids.csv'
val_ids_path = split_dir / 'val_ids.csv'
test_ids_path = split_dir / 'test_ids.csv'
split_manifest_path = split_dir / 'split_manifest.csv'
split_config_path = split_dir / 'split_config.json'

for path in [train_ids_path, val_ids_path, test_ids_path, split_manifest_path, split_config_path]:
    assert_generated_path_is_safe(path)

pd.DataFrame({'subject_id': train_ids}).to_csv(train_ids_path, index=False)
pd.DataFrame({'subject_id': val_ids}).to_csv(val_ids_path, index=False)
pd.DataFrame({'subject_id': test_ids}).to_csv(test_ids_path, index=False)
split_df.to_csv(split_manifest_path, index=False)

split_config = {
    'split_name': SPLIT_NAME,
    'split_seed': SPLIT_SEED,
    'train_fraction': 1.0 - VAL_FRACTION - TEST_FRACTION,
    'val_fraction': VAL_FRACTION,
    'test_fraction': TEST_FRACTION,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'official_split': 'train',
    'dev_split_column': 'dev_split',
    'stratification_column': 'lesion_volume_bin',
    'notes': (
        'Subject-level development train/val/test split created from public training cohort. '
        'Use test only for final internal held-out evaluation.'
    ),
}

split_config_path.write_text(json.dumps(split_config, indent=2), encoding='utf-8')

print(f'Split name: {SPLIT_NAME}')
print(f'Train subjects: {len(train_ids)}')
print(f'Val subjects:   {len(val_ids)}')
print(f'Test subjects:  {len(test_ids)}')

print('\nSplit balance by lesion volume bin:')
display(
    split_df
    .groupby(['lesion_volume_bin', 'dev_split'])
    .size()
    .reset_index(name='count')
    .sort_values(['lesion_volume_bin', 'dev_split'])
)

print('\nSaved:')
print(train_ids_path)
print(val_ids_path)
print(test_ids_path)
print(split_manifest_path)
print(split_config_path)


In [ ]:
config_path = GENERATED_DIRS['configs'] / 'preprocessing_v1_trial0.json'
config = json.loads(config_path.read_text(encoding='utf-8'))

config['split_name'] = SPLIT_NAME
config['split_policy'] = {
    'official_available_split': 'train only',
    'development_splits': ['train', 'val', 'test'],
    'test_usage': 'Internal held-out test. Do not use for checkpoint selection or iterative tuning.',
}

config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')

print(f'Updated config split_name to: {SPLIT_NAME}')


In [ ]:
import json
import numpy as np
import pandas as pd
import nibabel as nib
import torch
from pathlib import Path

PREPROCESSING_VERSION = 'v1_trial0'

preproc_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION
batch_preproc_dir = preproc_root / 'subjects_npz'
assert_generated_path_is_safe(batch_preproc_dir)
batch_preproc_dir.mkdir(parents=True, exist_ok=True)

config_path = GENERATED_DIRS['configs'] / f'preprocessing_{PREPROCESSING_VERSION}.json'
config = json.loads(config_path.read_text(encoding='utf-8'))

input_columns = config['input_channels']
target_column = config['target_column']
clip_ranges = config['clip_ranges']
target_spacing = tuple(config['spatial_plan']['target_spacing'])


preprocess_transform = build_preprocess_transform_clean_v2()

batch_rows = []

for idx, (_, row) in enumerate(manifest_df.iterrows(), start=1):
    sid = row['subject_id']
    out_path = batch_preproc_dir / f'{sid}_preprocessed_{PREPROCESSING_VERSION}.npz'
    assert_generated_path_is_safe(out_path)

    if out_path.exists():
        print(f'[{idx:03d}/{len(manifest_df)}] Skipping existing: {sid}')
    else:
        print(f'[{idx:03d}/{len(manifest_df)}] Preprocessing: {sid}')

        case = make_case_dict(row)
        out = preprocess_transform(case)

        image = out['image'].cpu().numpy().astype(np.float32)
        target = out['target'].cpu().numpy()
        target = (target > 0.5).astype(np.uint8)

        if image.shape[1:] != target.shape[1:]:
            raise RuntimeError(
                f'Shape mismatch for {sid}: image {image.shape}, target {target.shape}'
            )

        if np.isnan(image).any() or np.isinf(image).any():
            raise RuntimeError(f'NaN/Inf in preprocessed image for {sid}')

        np.savez_compressed(
            out_path,
            image=image,
            target=target,
            subject_id=sid,
            input_columns=np.array(input_columns),
            preprocessing_version=PREPROCESSING_VERSION,
        )

    arr = np.load(out_path, allow_pickle=True)
    image = arr['image']
    target = arr['target']

    batch_rows.append({
        'subject_id': sid,
        'output_path': str(out_path),
        'image_shape': tuple(image.shape),
        'target_shape': tuple(target.shape),
        'image_min': float(np.min(image)),
        'image_max': float(np.max(image)),
        'image_mean': float(np.mean(image)),
        'image_nan_count': int(np.isnan(image).sum()),
        'image_inf_count': int(np.isinf(image).sum()),
        'target_unique': np.unique(target).tolist(),
        'target_nonzero_voxels': int(np.count_nonzero(target)),
        'target_lost_after_resampling': (
            int(np.count_nonzero(target)) == 0
            and sid not in config['known_flags']['empty_target_masks']
        ),
    })

batch_preproc_df = pd.DataFrame(batch_rows)

batch_summary_path = preproc_root / 'batch_preprocessing_summary.csv'
assert_generated_path_is_safe(batch_summary_path)
batch_preproc_df.to_csv(batch_summary_path, index=False)

print('\nBatch preprocessing complete.')
print(f'Saved summary: {batch_summary_path}')

print('\nOutput count:')
print(len(batch_preproc_df))

print('\nTarget loss after resampling:')
display(batch_preproc_df[batch_preproc_df['target_lost_after_resampling']])

print('\nImage range summary:')
display(
    batch_preproc_df[['image_min', 'image_max', 'image_mean']]
    .describe()
)

print('\nShape counts:')
display(
    batch_preproc_df
    .groupby(['image_shape', 'target_shape'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)


In [ ]:
import json
import numpy as np
import pandas as pd
import nibabel as nib
import torch
from pathlib import Path

PREPROCESSING_VERSION = 'v1_trial0'

preproc_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION
batch_preproc_dir = preproc_root / 'subjects_npz'
assert_generated_path_is_safe(batch_preproc_dir)
batch_preproc_dir.mkdir(parents=True, exist_ok=True)

config_path = GENERATED_DIRS['configs'] / f'preprocessing_{PREPROCESSING_VERSION}.json'
config = json.loads(config_path.read_text(encoding='utf-8'))

input_columns = config['input_channels']
target_column = config['target_column']
clip_ranges = config['clip_ranges']
target_spacing = tuple(config['spatial_plan']['target_spacing'])


preprocess_transform = build_preprocess_transform_clean_v2()

batch_rows = []

for idx, (_, row) in enumerate(batch_manifest_df.iterrows(), start=1):
    sid = row['subject_id']
    out_path = batch_preproc_dir / f'{sid}_preprocessed_{PREPROCESSING_VERSION}.npz'
    assert_generated_path_is_safe(out_path)

    if out_path.exists():
        print(f'[{idx:03d}/{len(manifest_df)}] Skipping existing: {sid}')
    else:
        print(f'[{idx:03d}/{len(manifest_df)}] Preprocessing: {sid}')

        case = make_case_dict(row)
        out = preprocess_transform(case)

        image = out['image'].cpu().numpy().astype(np.float32)
        target = out['target'].cpu().numpy()
        target = (target > 0.5).astype(np.uint8)

        if image.shape[1:] != target.shape[1:]:
            raise RuntimeError(
                f'Shape mismatch for {sid}: image {image.shape}, target {target.shape}'
            )

        if np.isnan(image).any() or np.isinf(image).any():
            raise RuntimeError(f'NaN/Inf in preprocessed image for {sid}')

        np.savez_compressed(
            out_path,
            image=image,
            target=target,
            subject_id=sid,
            input_columns=np.array(input_columns),
            preprocessing_version=PREPROCESSING_VERSION,
        )

    arr = np.load(out_path, allow_pickle=True)
    image = arr['image']
    target = arr['target']

    batch_rows.append({
        'subject_id': sid,
        'output_path': str(out_path),
        'image_shape': tuple(image.shape),
        'target_shape': tuple(target.shape),
        'image_min': float(np.min(image)),
        'image_max': float(np.max(image)),
        'image_mean': float(np.mean(image)),
        'image_nan_count': int(np.isnan(image).sum()),
        'image_inf_count': int(np.isinf(image).sum()),
        'target_unique': np.unique(target).tolist(),
        'target_nonzero_voxels': int(np.count_nonzero(target)),
        'target_lost_after_resampling': (
            int(np.count_nonzero(target)) == 0
            and sid not in config['known_flags']['empty_target_masks']
        ),
    })

batch_preproc_df = pd.DataFrame(batch_rows)

batch_summary_path = preproc_root / 'batch_preprocessing_summary.csv'
assert_generated_path_is_safe(batch_summary_path)
batch_preproc_df.to_csv(batch_summary_path, index=False)

print('\nBatch preprocessing complete.')
print(f'Saved summary: {batch_summary_path}')

print('\nOutput count:')
print(len(batch_preproc_df))

print('\nTarget loss after resampling:')
display(batch_preproc_df[batch_preproc_df['target_lost_after_resampling']])

print('\nImage range summary:')
display(
    batch_preproc_df[['image_min', 'image_max', 'image_mean']]
    .describe()
)

print('\nShape counts:')
display(
    batch_preproc_df
    .groupby(['image_shape', 'target_shape'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)


In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path

FAILED_SID = 'sub-stroke0043'

row = manifest_df[manifest_df['subject_id'] == FAILED_SID].iloc[0]

diagnostic_columns = input_columns + [target_column]

print(f'Diagnosing source files for {FAILED_SID}')

diagnostic_rows = []

for col in diagnostic_columns:
    path = Path(row[col])
    print(f'\nChecking {col}:')
    print(path)

    try:
        img = nib.load(str(path))
        print(f'  header OK: shape={img.shape}, dtype={img.get_data_dtype()}')

        
        data = np.asanyarray(img.dataobj)
        finite_count = int(np.isfinite(data).sum())

        print(f'  full data read OK: finite voxels={finite_count}')

        diagnostic_rows.append({
            'subject_id': FAILED_SID,
            'path_column': col,
            'path': str(path),
            'header_ok': True,
            'full_read_ok': True,
            'error': None,
        })

    except Exception as exc:
        print(f'  FAILED: {repr(exc)}')

        diagnostic_rows.append({
            'subject_id': FAILED_SID,
            'path_column': col,
            'path': str(path),
            'header_ok': 'img' in locals(),
            'full_read_ok': False,
            'error': repr(exc),
        })

failed_file_diagnostic_df = pd.DataFrame(diagnostic_rows)
display(failed_file_diagnostic_df)

print('\nFailed files:')
display(failed_file_diagnostic_df[~failed_file_diagnostic_df['full_read_ok']])


In [ ]:
FAILED_SID = 'sub-stroke0043'
row = manifest_df[manifest_df['subject_id'] == FAILED_SID].iloc[0]

raw_cbf_path = Path(row['path_raw_cbf'])
deriv_cbf_path = Path(row['path_deriv_cbf'])
ncct_path = Path(row['path_raw_ncct'])

for label, path in {
    'raw_ncct': ncct_path,
    'raw_cbf': raw_cbf_path,
    'deriv_cbf_corrupt': deriv_cbf_path,
}.items():
    print(f'\n{label}: {path}')
    try:
        img = nib.load(str(path))
        data = np.asanyarray(img.dataobj)
        print(f'  shape: {img.shape}')
        print(f'  spacing: {img.header.get_zooms()[:3]}')
        print(f'  orientation: {"".join(nib.aff2axcodes(img.affine))}')
        print(f'  full read OK: finite voxels={np.isfinite(data).sum()}')
    except Exception as exc:
        print(f'  FAILED: {repr(exc)}')


In [ ]:
FAILED_SID = 'sub-stroke0043'

fallback_test_dir = preproc_root / 'fallback_tests'
assert_generated_path_is_safe(fallback_test_dir)
fallback_test_dir.mkdir(parents=True, exist_ok=True)

row = manifest_df[manifest_df['subject_id'] == FAILED_SID].iloc[0].copy()

print('Original derivative CBF:')
print(row['path_deriv_cbf'])

print('\nFallback raw CBF:')
print(row['path_raw_cbf'])

row['path_deriv_cbf'] = row['path_raw_cbf']

case = make_case_dict(row)

print(f'\nPreprocessing fallback test for {FAILED_SID}...')
out = preprocess_transform(case)

image = out['image'].cpu().numpy().astype(np.float32)
target = out['target'].cpu().numpy()
target = (target > 0.5).astype(np.uint8)

print(f'image shape:  {image.shape}')
print(f'target shape: {target.shape}')
print(f'image range:  {np.nanmin(image):.4f} to {np.nanmax(image):.4f}')
print(f'image NaNs:   {np.isnan(image).sum()}')
print(f'target voxels: {np.count_nonzero(target)}')

if image.shape[1:] != target.shape[1:]:
    raise RuntimeError('Fallback image/target shape mismatch.')

if np.isnan(image).any() or np.isinf(image).any():
    raise RuntimeError('Fallback image contains NaN/Inf.')

fallback_out_path = fallback_test_dir / f'{FAILED_SID}_fallback_raw_cbf_preprocessed_test.npz'
assert_generated_path_is_safe(fallback_out_path)

np.savez_compressed(
    fallback_out_path,
    image=image,
    target=target,
    subject_id=FAILED_SID,
    input_columns=np.array(input_columns),
    fallback_note='path_deriv_cbf replaced with path_raw_cbf because derivative CBF source file is truncated.',
)

print(f'Saved fallback test: {fallback_out_path}')


In [ ]:
EXCLUDED_SUBJECTS_PREPROCESSING = {
    'sub-stroke0043': (
        'Corrupt/truncated derivative CBF file: '
        'train/derivatives/sub-stroke0043/ses-01/perfusion-maps/'
        'sub-stroke0043_ses-01_space-ncct_cbf.nii.gz. '
        'Header loads but full gzip decompression fails with EOFError.'
    )
}

batch_manifest_df = manifest_df[
    ~manifest_df['subject_id'].isin(EXCLUDED_SUBJECTS_PREPROCESSING)
].copy()

print(f'Original subjects: {len(manifest_df)}')
print(f'Batch preprocessing subjects: {len(batch_manifest_df)}')
print('Excluded subjects:')
for sid, reason in EXCLUDED_SUBJECTS_PREPROCESSING.items():
    print(f'  {sid}: {reason}')


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PREPROCESSING_VERSION = 'v1_trial0'
preproc_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION
batch_preproc_dir = preproc_root / 'subjects_npz'

expected_subjects = set(manifest_df['subject_id'])
found_files = sorted(batch_preproc_dir.glob(f'*_preprocessed_{PREPROCESSING_VERSION}.npz'))

audit_rows = []

for path in found_files:
    sid = path.name.split('_preprocessed_')[0]

    try:
        arr = np.load(path, allow_pickle=True)
        image = arr['image']
        target = arr['target']

        audit_rows.append({
            'subject_id': sid,
            'path': str(path),
            'readable': True,
            'image_shape': tuple(image.shape),
            'target_shape': tuple(target.shape),
            'shape_match': image.shape[1:] == target.shape[1:],
            'image_channel_count': image.shape[0],
            'image_nan_count': int(np.isnan(image).sum()),
            'image_inf_count': int(np.isinf(image).sum()),
            'image_min': float(np.min(image)),
            'image_max': float(np.max(image)),
            'target_unique': np.unique(target).tolist(),
            'target_nonzero_voxels': int(np.count_nonzero(target)),
            'error': None,
        })

    except Exception as exc:
        audit_rows.append({
            'subject_id': sid,
            'path': str(path),
            'readable': False,
            'image_shape': None,
            'target_shape': None,
            'shape_match': False,
            'image_channel_count': None,
            'image_nan_count': None,
            'image_inf_count': None,
            'image_min': None,
            'image_max': None,
            'target_unique': None,
            'target_nonzero_voxels': None,
            'error': repr(exc),
        })

audit_df = pd.DataFrame(audit_rows)

missing_subjects = sorted(expected_subjects - set(audit_df['subject_id']))

audit_path = preproc_root / 'preprocessed_npz_audit.csv'
assert_generated_path_is_safe(audit_path)
audit_df.to_csv(audit_path, index=False)

print(f'Expected subjects: {len(expected_subjects)}')
print(f'Found NPZ files:   {len(found_files)}')
print(f'Missing subjects:  {len(missing_subjects)}')
print(f'Saved audit: {audit_path}')

print('\nUnreadable / invalid files:')
display(
    audit_df[
        (~audit_df['readable'])
        | (~audit_df['shape_match'])
        | (audit_df['image_channel_count'] != 6)
        | (audit_df['image_nan_count'] > 0)
        | (audit_df['image_inf_count'] > 0)
        | (audit_df['image_min'] < 0)
        | (audit_df['image_max'] > 1)
    ]
)

print('\nTarget voxel summary after preprocessing:')
display(audit_df['target_nonzero_voxels'].describe())

print('\nSubjects with empty target after preprocessing:')
display(
    audit_df[audit_df['target_nonzero_voxels'] == 0]
    .sort_values('subject_id')
)


In [ ]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'

preproc_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION
audit_path = preproc_root / 'preprocessed_npz_audit.csv'
split_manifest_path = GENERATED_DIRS['splits'] / SPLIT_NAME / 'split_manifest.csv'

audit_df = pd.read_csv(audit_path)
split_df = pd.read_csv(split_manifest_path)

EXCLUDED_FROM_TRIAL0_MODELING = {
    'sub-stroke0043': (
        'No preprocessed NPZ because derivative CBF source file is corrupt/truncated.'
    )
}

FLAGGED_FOR_TRIAL0_MODELING = {
    'sub-stroke0171': (
        'Original target had 3 voxels; target became empty after v1 resampling.'
    )
}

valid_audit_df = audit_df[
    (audit_df['readable'])
    & (audit_df['shape_match'])
    & (audit_df['image_channel_count'] == 6)
    & (audit_df['image_nan_count'] == 0)
    & (audit_df['image_inf_count'] == 0)
    & (audit_df['image_min'] >= 0)
    & (audit_df['image_max'] <= 1)
].copy()

model_manifest_df = split_df.merge(
    valid_audit_df[
        [
            'subject_id',
            'path',
            'image_shape',
            'target_shape',
            'target_nonzero_voxels',
        ]
    ],
    on='subject_id',
    how='inner',
)

model_manifest_df = model_manifest_df.rename(columns={
    'path': 'preprocessed_npz_path',
    'target_nonzero_voxels': 'preprocessed_target_nonzero_voxels',
})

model_manifest_df['excluded_from_trial0_modeling'] = model_manifest_df['subject_id'].isin(
    EXCLUDED_FROM_TRIAL0_MODELING
)
model_manifest_df['trial0_flag'] = model_manifest_df['subject_id'].map(
    FLAGGED_FOR_TRIAL0_MODELING
)

missing_from_model_manifest = sorted(set(split_df['subject_id']) - set(model_manifest_df['subject_id']))

model_manifest_path = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
model_exclusion_path = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0_exclusions.json'

assert_generated_path_is_safe(model_manifest_path)
assert_generated_path_is_safe(model_exclusion_path)

model_manifest_df.to_csv(model_manifest_path, index=False)

exclusion_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'excluded_from_trial0_modeling': EXCLUDED_FROM_TRIAL0_MODELING,
    'flagged_for_trial0_modeling': FLAGGED_FOR_TRIAL0_MODELING,
    'missing_from_model_manifest': missing_from_model_manifest,
}

model_exclusion_path.write_text(json.dumps(exclusion_record, indent=2), encoding='utf-8')

print(f'Model manifest rows: {len(model_manifest_df)}')
print(f'Saved model manifest: {model_manifest_path}')
print(f'Saved exclusions/flags: {model_exclusion_path}')

print('\nMissing from model manifest:')
print(missing_from_model_manifest)

print('\nModel split counts:')
display(
    model_manifest_df
    .groupby('dev_split')
    .size()
    .reset_index(name='count')
)

print('\nModel split counts by lesion volume bin:')
display(
    model_manifest_df
    .groupby(['lesion_volume_bin', 'dev_split'])
    .size()
    .reset_index(name='count')
    .sort_values(['lesion_volume_bin', 'dev_split'])
)

print('\nFlagged subjects present in model manifest:')
display(
    model_manifest_df[model_manifest_df['trial0_flag'].notna()]
)


In [ ]:
from google.colab import drive
drive.mount('/mnt/storage')


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path('/path/to/project')
RAW_DATA_ROOT = PROJECT_ROOT / 'ISLES24 dataset'
TRAIN_ROOT = RAW_DATA_ROOT / 'train'

RAW_IMAGE_ROOT = TRAIN_ROOT / 'raw_data'
DERIVATIVES_ROOT = TRAIN_ROOT / 'derivatives'
PHENOTYPE_ROOT = TRAIN_ROOT / 'phenotype'

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
    'configs': PROJECT_ROOT / 'configs',
}

def resolved(path: Path) -> Path:
    return path.expanduser().resolve(strict=False)

def is_inside(child: Path, parent: Path) -> bool:
    try:
        resolved(child).relative_to(resolved(parent))
        return True
    except ValueError:
        return False

def assert_generated_path_is_safe(path: Path) -> None:
    if not is_inside(path, PROJECT_ROOT):
        raise ValueError(f'Path is outside PROJECT_ROOT: {path}')
    if is_inside(path, RAW_DATA_ROOT):
        raise ValueError(f'Generated path is inside RAW_DATA_ROOT, forbidden: {path}')

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'

manifest_path = GENERATED_DIRS['metadata'] / 'subject_manifest_paths_only_trial0.csv'
split_manifest_path = GENERATED_DIRS['splits'] / SPLIT_NAME / 'split_manifest.csv'
config_path = GENERATED_DIRS['configs'] / f'preprocessing_{PREPROCESSING_VERSION}.json'

manifest_df = pd.read_csv(manifest_path)
split_df = pd.read_csv(split_manifest_path)
config = json.loads(config_path.read_text(encoding='utf-8'))

input_columns = config['input_channels']
target_column = config['target_column']
clip_ranges = config['clip_ranges']
target_spacing = tuple(config['spatial_plan']['target_spacing'])

preproc_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION
batch_preproc_dir = preproc_root / 'subjects_npz'

existing_npz_count = len(list(batch_preproc_dir.glob('*.npz')))

print('Resume setup loaded.')
print(f'Manifest rows: {len(manifest_df)}')
print(f'Split rows: {len(split_df)}')
print(f'Preprocessed dir: {batch_preproc_dir}')
print(f'Existing NPZ files: {existing_npz_count}')


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference

TRAINING_RUN_NAME = 'trial0_unet_baseline_v1'
MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'

PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 512
BATCH_SIZE = 1
NUM_WORKERS = 0
SEED = 20240824

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')

model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

train_df = model_manifest_df[model_manifest_df['dev_split'] == 'train'].reset_index(drop=True)
val_df = model_manifest_df[model_manifest_df['dev_split'] == 'val'].reset_index(drop=True)
test_df = model_manifest_df[model_manifest_df['dev_split'] == 'test'].reset_index(drop=True)

print(f'Train subjects: {len(train_df)}')
print(f'Val subjects:   {len(val_df)}')
print(f'Test subjects:  {len(test_df)}')

class ISLESPreprocessedPatchDataset(Dataset):
    def __init__(
        self,
        df,
        patch_size=(128, 128, 64),
        samples_per_epoch=None,
        positive_prob=0.70,
        seed=20240824,
    ):
        self.df = df.reset_index(drop=True)
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch or len(self.df)
        self.positive_prob = positive_prob
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.samples_per_epoch

    def _load_case(self, index):
        row = self.df.iloc[index % len(self.df)]
        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)
        return row['subject_id'], image, target

    def _pad_to_patch(self, image, target):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            pad_spec_img = (
                (0, 0),
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )
            pad_spec_tgt = (
                (0, 0),
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )
            image = np.pad(image, pad_spec_img, mode='constant', constant_values=0)
            target = np.pad(target, pad_spec_tgt, mode='constant', constant_values=0)

        return image, target

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            center = coords[self.rng.integers(0, len(coords))]
            ch, cw, cd = center
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        subject_id, image, target = self._load_case(index)
        image, target = self._pad_to_patch(image, target)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        image_patch = image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]
        target_patch = target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]

        return {
            'image': torch.from_numpy(image_patch),
            'target': torch.from_numpy(target_patch),
            'subject_id': subject_id,
        }

train_dataset = ISLESPreprocessedPatchDataset(
    train_df,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=0.70,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

batch = next(iter(train_loader))
print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch image range:', float(batch['image'].min()), float(batch['image'].max()))
print('Batch target voxels:', int(batch['target'].sum()))


In [ ]:
import json
import torch
import torch.nn as nn
from pathlib import Path
from monai.networks.nets import UNet
from monai.losses import DiceCELoss

MAX_EPOCHS = 60
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
VAL_EVERY = 1
SAVE_EVERY_EPOCH = True


model = UNet(
    spatial_dims=3,
    in_channels=6,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2,
    norm='INSTANCE',
).to(device)

loss_fn = DiceCELoss(
    sigmoid=True,
    squared_pred=True,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model trainable parameters: {count_parameters(model):,}')

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'model': 'MONAI UNet compact 3D baseline',
    'in_channels': 6,
    'out_channels': 1,
    'channels': [16, 32, 64, 128],
    'strides': [2, 2, 2],
    'num_res_units': 2,
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'val_every': VAL_EVERY,
    'loss': 'DiceCELoss(sigmoid=True, squared_pred=True)',
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR',
    'seed': SEED,
    'excluded_subjects': ['sub-stroke0043'],
    'flagged_subjects': ['sub-stroke0171'],
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

history = []
best_val_dice = -1.0
start_epoch = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming from checkpoint: {latest_checkpoint_path}')
    print(f'Start epoch: {start_epoch}')
    print(f'Best val Dice so far: {best_val_dice:.4f}')
else:
    print('Starting new training run.')
    print(f'Checkpoints will be saved to: {checkpoint_dir}')


In [ ]:
import numpy as np
import torch
import pandas as pd
from monai.inferers import sliding_window_inference

VAL_PATCH_SIZE = PATCH_SIZE
VAL_SW_BATCH_SIZE = 1
VAL_THRESHOLD = 0.5

def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    return image, target

def pad_to_roi_size(image, target, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        pad_spec_img = (
            (0, 0),
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )
        pad_spec_tgt = (
            (0, 0),
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )

        image = np.pad(image, pad_spec_img, mode='constant', constant_values=0)
        target = np.pad(target, pad_spec_tgt, mode='constant', constant_values=0)

    return image, target

@torch.no_grad()
def validate_full_volume(model, val_df, max_subjects=None):
    model.eval()

    dice_rows = []
    eval_df = val_df if max_subjects is None else val_df.head(max_subjects)

    for eval_idx, (_, row) in enumerate(eval_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        image, target = pad_to_roi_size(image, target, VAL_PATCH_SIZE)

        image_t = torch.from_numpy(image[None]).to(device)

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=VAL_PATCH_SIZE,
            sw_batch_size=VAL_SW_BATCH_SIZE,
            predictor=model,
            overlap=0.5,
        )

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred = prob >= VAL_THRESHOLD

        target_np = target[0] > 0
        dice = dice_score_binary(pred, target_np)

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())

        dice_rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
        })

        print(
            f'  Val {eval_idx}/{len(eval_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}'
        )

    dice_df = pd.DataFrame(dice_rows)

    mean_dice_all = float(dice_df['dice'].mean()) if len(dice_df) else 0.0

    nonempty_df = dice_df[~dice_df['target_empty']]
    mean_dice_nonempty = float(nonempty_df['dice'].mean()) if len(nonempty_df) else 0.0

    summary = {
        'mean_dice_all': mean_dice_all,
        'mean_dice_nonempty': mean_dice_nonempty,
        'num_cases': int(len(dice_df)),
        'num_empty_targets': int(dice_df['target_empty'].sum()) if len(dice_df) else 0,
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()) if len(dice_df) else 0.0,
    }

    return summary, dice_df


In [ ]:
import time
import pandas as pd
import numpy as np
import torch

EARLY_STOPPING_PATIENCE = 12
MIN_DELTA = 1e-4


if 'epochs_without_improvement' not in globals():
    epochs_without_improvement = 0

def save_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch(epoch):
    model.train()

    epoch_losses = []
    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = loss_fn(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = float(loss.detach().cpu())
        epoch_losses.append(loss_value)

        if step == 1 or step % 50 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | '
                f'Step {step:04d}/{len(train_loader)} | '
                f'Loss {loss_value:.4f}'
            )

    mean_loss = float(np.mean(epoch_losses))
    elapsed = time.time() - start_time

    return mean_loss, elapsed

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_loss, elapsed = train_one_epoch(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning validation...')
        val_summary, val_df_epoch = validate_full_volume(model, val_df)

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'lr': current_lr,
        'elapsed_seconds': elapsed,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Epoch {epoch} complete.')
    print(f'Train loss: {train_loss:.4f}')
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'LR: {current_lr:.8f}')
    print(f'Elapsed: {elapsed / 60:.2f} min')
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after '
            f'{epochs_without_improvement} epochs without validation improvement.'
        )
        break

print('Training complete.')


In [ ]:
import pandas as pd
import torch
from pathlib import Path

history_path = log_dir / 'training_history.csv'
history_df = pd.read_csv(history_path)

display(history_df.tail(15))

best_row = history_df.loc[history_df['val_dice_nonempty'].idxmax()]

print('Best validation epoch:')
display(best_row)

print(f"Best non-empty val Dice: {best_row['val_dice_nonempty']:.4f}")
print(f"All-case val Dice at best epoch: {best_row['val_dice_all']:.4f}")
print(f"Train loss at best epoch: {best_row['train_loss']:.4f}")
print(f"Best checkpoint path: {best_checkpoint_path}")
print(f"Latest checkpoint path: {latest_checkpoint_path}")

best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
print(f"Best checkpoint stored epoch: {best_checkpoint['epoch']}")
print(f"Best checkpoint stored Dice: {best_checkpoint['best_val_dice']:.4f}")


In [ ]:
import numpy as np
import pandas as pd
import torch


best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best checkpoint epoch: {best_checkpoint['epoch']}")
print(f"Stored best val Dice: {best_checkpoint['best_val_dice']:.4f}")

THRESHOLDS = np.round(np.arange(0.05, 0.76, 0.05), 2)

@torch.no_grad()
def validation_threshold_sweep(model, val_df, thresholds=THRESHOLDS, max_subjects=None):
    model.eval()

    rows = []
    eval_df = val_df if max_subjects is None else val_df.head(max_subjects)

    for eval_idx, (_, row) in enumerate(eval_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        image, target = pad_to_roi_size(image, target, VAL_PATCH_SIZE)

        image_t = torch.from_numpy(image[None]).to(device)

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=VAL_PATCH_SIZE,
            sw_batch_size=VAL_SW_BATCH_SIZE,
            predictor=model,
            overlap=0.5,
        )

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        target_np = target[0] > 0

        target_voxels = int(target_np.sum())

        for threshold in thresholds:
            pred = prob >= threshold
            pred_voxels = int(pred.sum())
            dice = dice_score_binary(pred, target_np)

            rows.append({
                'subject_id': sid,
                'threshold': float(threshold),
                'dice': dice,
                'target_voxels': target_voxels,
                'pred_voxels': pred_voxels,
                'target_empty': target_voxels == 0,
                'pred_empty': pred_voxels == 0,
            })

        print(f'Completed {eval_idx}/{len(eval_df)}: {sid}')

        del image_t, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    sweep_df = pd.DataFrame(rows)

    summary_df = (
        sweep_df
        .groupby('threshold')
        .apply(lambda g: pd.Series({
            'mean_dice_all': g['dice'].mean(),
            'mean_dice_nonempty': g.loc[~g['target_empty'], 'dice'].mean(),
            'num_empty_targets': int(g['target_empty'].sum()),
            'mean_pred_voxels': g['pred_voxels'].mean(),
            'median_pred_voxels': g['pred_voxels'].median(),
            'num_zero_predictions': int(g['pred_empty'].sum()),
        }))
        .reset_index()
        .sort_values('mean_dice_nonempty', ascending=False)
    )

    return sweep_df, summary_df

sweep_df, threshold_summary_df = validation_threshold_sweep(model, val_df)

sweep_detail_path = log_dir / 'best_checkpoint_validation_threshold_sweep_cases.csv'
sweep_summary_path = log_dir / 'best_checkpoint_validation_threshold_sweep_summary.csv'

assert_generated_path_is_safe(sweep_detail_path)
assert_generated_path_is_safe(sweep_summary_path)

sweep_df.to_csv(sweep_detail_path, index=False)
threshold_summary_df.to_csv(sweep_summary_path, index=False)

print('Best thresholds by non-empty validation Dice:')
display(threshold_summary_df.head(10))

print(f'Saved case-level sweep: {sweep_detail_path}')
print(f'Saved threshold summary: {sweep_summary_path}')


In [ ]:
import pandas as pd

sweep_df = pd.read_csv(log_dir / 'best_checkpoint_validation_threshold_sweep_cases.csv')

case_best_df = (
    sweep_df
    .sort_values(['subject_id', 'dice'], ascending=[True, False])
    .groupby('subject_id')
    .head(1)
    .reset_index(drop=True)
    .sort_values('dice', ascending=False)
)

print('Best per-case validation Dice after threshold sweep:')
display(case_best_df)

print('\nSummary by lesion size:')
case_best_with_bins_df = case_best_df.merge(
    val_df[['subject_id', 'lesion_volume_bin']],
    on='subject_id',
    how='left'
)

display(
    case_best_with_bins_df
    .groupby('lesion_volume_bin')
    .agg(
        cases=('subject_id', 'count'),
        mean_best_dice=('dice', 'mean'),
        median_best_dice=('dice', 'median'),
        mean_target_voxels=('target_voxels', 'mean'),
        mean_pred_voxels=('pred_voxels', 'mean'),
    )
    .reset_index()
    .sort_values('mean_best_dice', ascending=False)
)

print('\nWorst 10 cases:')
display(case_best_with_bins_df.sort_values('dice').head(10))

case_best_path = log_dir / 'best_checkpoint_per_case_best_threshold.csv'
assert_generated_path_is_safe(case_best_path)
case_best_with_bins_df.to_csv(case_best_path, index=False)

print(f'Saved per-case threshold summary: {case_best_path}')


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

visual_qa_dir = log_dir / 'visual_qa_best_checkpoint'
assert_generated_path_is_safe(visual_qa_dir)
visual_qa_dir.mkdir(parents=True, exist_ok=True)

CHANNEL_NAMES = ['NCCT', 'CTA', 'CBF', 'CBV', 'MTT', 'Tmax']

cases_to_inspect = [
    'sub-stroke0070', 
    'sub-stroke0080', 
    'sub-stroke0096', 
    'sub-stroke0114', 
    'sub-stroke0094', 
    'sub-stroke0057', 
    'sub-stroke0004', 
    'sub-stroke0184', 
]

case_best_df = pd.read_csv(log_dir / 'best_checkpoint_per_case_best_threshold.csv')

@torch.no_grad()
def predict_case_probability(npz_path):
    image, target = load_full_case_npz(npz_path)
    image, target = pad_to_roi_size(image, target, VAL_PATCH_SIZE)

    image_t = torch.from_numpy(image[None]).to(device)

    logits = sliding_window_inference(
        inputs=image_t,
        roi_size=VAL_PATCH_SIZE,
        sw_batch_size=VAL_SW_BATCH_SIZE,
        predictor=model,
        overlap=0.5,
    )

    prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

    del image_t, logits
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return image, target[0] > 0, prob

def choose_slice(target, prob):
    target_per_slice = target.sum(axis=(0, 1))
    if target_per_slice.max() > 0:
        return int(np.argmax(target_per_slice))

    prob_per_slice = prob.sum(axis=(0, 1))
    return int(np.argmax(prob_per_slice))

def overlay_mask(base, mask, color):
    overlay = np.zeros((*base.shape, 4), dtype=np.float32)
    overlay[..., :3] = color
    overlay[..., 3] = mask.astype(np.float32) * 0.45
    return overlay

qa_rows = []

for sid in cases_to_inspect:
    row = val_df[val_df['subject_id'] == sid].iloc[0]
    best_case = case_best_df[case_best_df['subject_id'] == sid].iloc[0]
    threshold = float(best_case['threshold'])

    image, target, prob = predict_case_probability(row['preprocessed_npz_path'])
    pred = prob >= threshold

    z = choose_slice(target, prob)

    target_voxels = int(target.sum())
    pred_voxels = int(pred.sum())
    max_prob = float(prob.max())
    p99_prob = float(np.percentile(prob, 99))
    p999_prob = float(np.percentile(prob, 99.9))

    qa_rows.append({
        'subject_id': sid,
        'lesion_volume_bin': row['lesion_volume_bin'],
        'best_threshold': threshold,
        'best_dice': float(best_case['dice']),
        'target_voxels': target_voxels,
        'pred_voxels': pred_voxels,
        'max_prob': max_prob,
        'p99_prob': p99_prob,
        'p999_prob': p999_prob,
        'display_slice_z': z,
    })

    ncct = image[0, :, :, z]
    cbf = image[2, :, :, z]
    tmax = image[5, :, :, z]
    prob_z = prob[:, :, z]
    target_z = target[:, :, z]
    pred_z = pred[:, :, z]

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    axes[0].imshow(ncct.T, cmap='gray', origin='lower')
    axes[0].imshow(overlay_mask(ncct.T, target_z.T, [1, 0, 0]), origin='lower')
    axes[0].imshow(overlay_mask(ncct.T, pred_z.T, [0, 1, 1]), origin='lower')
    axes[0].set_title('NCCT overlay\nred=target, cyan=pred')

    axes[1].imshow(cbf.T, cmap='gray', origin='lower')
    axes[1].set_title('CBF')

    axes[2].imshow(tmax.T, cmap='gray', origin='lower')
    axes[2].set_title('Tmax')

    im = axes[3].imshow(prob_z.T, cmap='magma', origin='lower', vmin=0, vmax=1)
    axes[3].set_title('Predicted probability')
    fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    for ax in axes:
        ax.axis('off')

    fig.suptitle(
        f'{sid} | bin={row["lesion_volume_bin"]} | '
        f'Dice={best_case["dice"]:.4f} | thr={threshold:.2f} | z={z}',
        fontsize=14
    )

    out_path = visual_qa_dir / f'{sid}_best_checkpoint_visual_qa.png'
    assert_generated_path_is_safe(out_path)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

qa_df = pd.DataFrame(qa_rows)
qa_path = visual_qa_dir / 'visual_qa_probability_summary.csv'
assert_generated_path_is_safe(qa_path)
qa_df.to_csv(qa_path, index=False)

display(qa_df)
print(f'Saved visual QA images and summary under: {visual_qa_dir}')


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import json
import random
import numpy as np
import pandas as pd

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()

    if path != project_root and project_root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')

print(f'Project root: {PROJECT_ROOT}')
print('Generated directories ready.')


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from monai.networks.nets import UNet
from monai.losses import DiceFocalLoss
from monai.inferers import sliding_window_inference

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'
TRAINING_RUN_NAME = 'trial1_unet_imbalance_tuned_v1'

SEED = 20240824

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')


In [ ]:
MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'

model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

train_df = model_manifest_df[
    model_manifest_df['dev_split'] == 'train'
].reset_index(drop=True)

val_df = model_manifest_df[
    model_manifest_df['dev_split'] == 'val'
].reset_index(drop=True)

test_df = model_manifest_df[
    model_manifest_df['dev_split'] == 'test'
].reset_index(drop=True)

train_nonempty_df = train_df[
    train_df['preprocessed_target_nonzero_voxels'] > 0
].reset_index(drop=True)

print(f'Total manifest subjects: {len(model_manifest_df)}')
print(f'Train subjects: {len(train_df)}')
print(f'Non-empty train subjects for Trial1: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')

print('\nExcluded empty train subjects for Trial1:')
print(sorted(set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])))


In [ ]:
PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1024
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.85

MAX_EPOCHS = 80
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
VAL_EVERY = 1
VAL_THRESHOLD = 0.20
EARLY_STOPPING_PATIENCE = 15
MIN_DELTA = 1e-4

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

class ISLESPreprocessedPatchDataset(Dataset):
    def __init__(
        self,
        df,
        patch_size=(128, 128, 64),
        samples_per_epoch=None,
        positive_prob=0.85,
        seed=20240824,
    ):
        self.df = df.reset_index(drop=True)
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch or len(self.df)
        self.positive_prob = positive_prob
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.samples_per_epoch

    def _load_case(self, index):
        row = self.df.iloc[index % len(self.df)]
        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)
        return row['subject_id'], image, target

    def _pad_to_patch(self, image, target):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            pad_spec = (
                (0, 0),
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )
            image = np.pad(image, pad_spec, mode='constant', constant_values=0)
            target = np.pad(target, pad_spec, mode='constant', constant_values=0)

        return image, target

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            center = coords[self.rng.integers(0, len(coords))]
            ch, cw, cd = center
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        subject_id, image, target = self._load_case(index)
        image, target = self._pad_to_patch(image, target)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'subject_id': subject_id,
        }

train_dataset = ISLESPreprocessedPatchDataset(
    train_nonempty_df,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = UNet(
    spatial_dims=3,
    in_channels=6,
    out_channels=1,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=2,
    norm='INSTANCE',
).to(device)

loss_fn = DiceFocalLoss(
    sigmoid=True,
    squared_pred=True,
    lambda_dice=1.0,
    lambda_focal=1.0,
    gamma=2.0,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'model': 'MONAI UNet compact 3D, imbalance-tuned',
    'trial_note': 'CNN-only Trial1 after Trial0 showed small-lesion failure but visually acceptable preprocessing.',
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'train_subjects': len(train_nonempty_df),
    'excluded_empty_train_subjects': sorted(
        set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])
    ),
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'validation_threshold': VAL_THRESHOLD,
    'loss': 'DiceFocalLoss(sigmoid=True, gamma=2.0)',
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR',
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'seed': SEED,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial1 from: {latest_checkpoint_path}')
    print(f'Start epoch: {start_epoch}')
    print(f'Best val Dice so far: {best_val_dice:.4f}')
    print(f'Epochs without improvement: {epochs_without_improvement}')
else:
    print('Starting new Trial1 run.')

batch = next(iter(train_loader))
print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))

print(f'Model trainable parameters: {count_parameters(model):,}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
import numpy as np
import pandas as pd
import torch
from monai.inferers import sliding_window_inference

VAL_PATCH_SIZE = PATCH_SIZE
VAL_SW_BATCH_SIZE = 1

def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    return image, target

def pad_to_roi_size(image, target, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        pad_spec = (
            (0, 0),
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )
        image = np.pad(image, pad_spec, mode='constant', constant_values=0)
        target = np.pad(target, pad_spec, mode='constant', constant_values=0)

    return image, target

@torch.no_grad()
def validate_full_volume(model, val_df, max_subjects=None):
    model.eval()

    dice_rows = []
    eval_df = val_df if max_subjects is None else val_df.head(max_subjects)

    for eval_idx, (_, row) in enumerate(eval_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        image, target = pad_to_roi_size(image, target, VAL_PATCH_SIZE)

        image_t = torch.from_numpy(image[None]).to(device)

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=VAL_PATCH_SIZE,
            sw_batch_size=VAL_SW_BATCH_SIZE,
            predictor=model,
            overlap=0.5,
        )

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred = prob >= VAL_THRESHOLD

        target_np = target[0] > 0
        dice = dice_score_binary(pred, target_np)

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())

        dice_rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
        })

        print(
            f'  Val {eval_idx}/{len(eval_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}'
        )

        del image_t, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    dice_df = pd.DataFrame(dice_rows)

    mean_dice_all = float(dice_df['dice'].mean()) if len(dice_df) else 0.0
    nonempty_df = dice_df[~dice_df['target_empty']]
    mean_dice_nonempty = float(nonempty_df['dice'].mean()) if len(nonempty_df) else 0.0

    summary = {
        'mean_dice_all': mean_dice_all,
        'mean_dice_nonempty': mean_dice_nonempty,
        'num_cases': int(len(dice_df)),
        'num_empty_targets': int(dice_df['target_empty'].sum()) if len(dice_df) else 0,
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()) if len(dice_df) else 0.0,
    }

    return summary, dice_df


In [ ]:
import time
import numpy as np
import pandas as pd
import torch

def save_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch(epoch):
    model.train()

    epoch_losses = []
    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type='cuda',
            enabled=torch.cuda.is_available(),
        ):
            logits = model(images)
            loss = loss_fn(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = float(loss.detach().cpu())
        epoch_losses.append(loss_value)

        if step == 1 or step % 50 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | '
                f'Step {step:04d}/{len(train_loader)} | '
                f'Loss {loss_value:.4f}'
            )

    mean_loss = float(np.mean(epoch_losses))
    elapsed = time.time() - start_time

    return mean_loss, elapsed

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'Trial1 Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_loss, elapsed = train_one_epoch(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning validation...')
        val_summary, val_df_epoch = validate_full_volume(model, val_df)

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'lr': current_lr,
        'elapsed_seconds': elapsed,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Trial1 epoch {epoch} complete.')
    print(f'Train loss: {train_loss:.4f}')
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'LR: {current_lr:.8f}')
    print(f'Elapsed: {elapsed / 60:.2f} min')
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after '
            f'{epochs_without_improvement} epochs without validation improvement.'
        )
        break

print('Trial1 training complete.')


In [ ]:
import numpy as np
import pandas as pd
import torch

best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print(f"Loaded Trial1 best checkpoint epoch: {best_checkpoint['epoch']}")
print(f"Stored best val Dice: {best_checkpoint['best_val_dice']:.4f}")

THRESHOLDS = np.round(np.arange(0.05, 0.76, 0.05), 2)

@torch.no_grad()
def validation_threshold_sweep(model, val_df, thresholds=THRESHOLDS):
    model.eval()
    rows = []

    for eval_idx, (_, row) in enumerate(val_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        image, target = pad_to_roi_size(image, target, VAL_PATCH_SIZE)

        image_t = torch.from_numpy(image[None]).to(device)

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=VAL_PATCH_SIZE,
            sw_batch_size=VAL_SW_BATCH_SIZE,
            predictor=model,
            overlap=0.5,
        )

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        target_np = target[0] > 0
        target_voxels = int(target_np.sum())

        for threshold in thresholds:
            pred = prob >= threshold
            rows.append({
                'subject_id': sid,
                'threshold': float(threshold),
                'dice': dice_score_binary(pred, target_np),
                'target_voxels': target_voxels,
                'pred_voxels': int(pred.sum()),
                'target_empty': target_voxels == 0,
                'pred_empty': int(pred.sum()) == 0,
            })

        print(f'Completed {eval_idx}/{len(val_df)}: {sid}')

        del image_t, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    sweep_df = pd.DataFrame(rows)

    summary_df = (
        sweep_df
        .groupby('threshold')
        .apply(lambda g: pd.Series({
            'mean_dice_all': g['dice'].mean(),
            'mean_dice_nonempty': g.loc[~g['target_empty'], 'dice'].mean(),
            'num_empty_targets': int(g['target_empty'].sum()),
            'mean_pred_voxels': g['pred_voxels'].mean(),
            'median_pred_voxels': g['pred_voxels'].median(),
            'num_zero_predictions': int(g['pred_empty'].sum()),
        }))
        .reset_index()
        .sort_values('mean_dice_nonempty', ascending=False)
    )

    return sweep_df, summary_df

sweep_df, threshold_summary_df = validation_threshold_sweep(model, val_df)

sweep_detail_path = log_dir / 'trial1_best_checkpoint_threshold_sweep_cases.csv'
sweep_summary_path = log_dir / 'trial1_best_checkpoint_threshold_sweep_summary.csv'

assert_generated_path_is_safe(sweep_detail_path)
assert_generated_path_is_safe(sweep_summary_path)

sweep_df.to_csv(sweep_detail_path, index=False)
threshold_summary_df.to_csv(sweep_summary_path, index=False)

print('Best Trial1 thresholds by non-empty validation Dice:')
display(threshold_summary_df.head(10))

print(f'Saved case-level sweep: {sweep_detail_path}')
print(f'Saved threshold summary: {sweep_summary_path}')


In [ ]:
import pandas as pd

trial1_sweep_df = pd.read_csv(
    log_dir / 'trial1_best_checkpoint_threshold_sweep_cases.csv'
)

trial1_case_best_df = (
    trial1_sweep_df
    .sort_values(['subject_id', 'dice'], ascending=[True, False])
    .groupby('subject_id')
    .head(1)
    .reset_index(drop=True)
    .sort_values('dice', ascending=False)
)

trial1_case_best_with_bins_df = trial1_case_best_df.merge(
    val_df[['subject_id', 'lesion_volume_bin']],
    on='subject_id',
    how='left',
)

print('Trial1 best per-case validation Dice after threshold sweep:')
display(trial1_case_best_with_bins_df)

print('\nTrial1 summary by lesion size:')
display(
    trial1_case_best_with_bins_df
    .groupby('lesion_volume_bin')
    .agg(
        cases=('subject_id', 'count'),
        mean_best_dice=('dice', 'mean'),
        median_best_dice=('dice', 'median'),
        mean_target_voxels=('target_voxels', 'mean'),
        mean_pred_voxels=('pred_voxels', 'mean'),
    )
    .reset_index()
    .sort_values('mean_best_dice', ascending=False)
)

trial1_case_best_path = log_dir / 'trial1_best_checkpoint_per_case_best_threshold.csv'
assert_generated_path_is_safe(trial1_case_best_path)
trial1_case_best_with_bins_df.to_csv(trial1_case_best_path, index=False)

print(f'Saved Trial1 per-case best summary: {trial1_case_best_path}')


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import json
import random
import numpy as np
import pandas as pd

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()

    if path != project_root and project_root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')

print(f'Project root: {PROJECT_ROOT}')
print('Generated directories ready.')


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from monai.inferers import sliding_window_inference

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'
TRAINING_RUN_NAME = 'trial2_segresnet_imbalance_tuned_v1'

SEED = 20240824

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')


In [ ]:
MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'

model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

train_df = model_manifest_df[
    model_manifest_df['dev_split'] == 'train'
].reset_index(drop=True)

val_df = model_manifest_df[
    model_manifest_df['dev_split'] == 'val'
].reset_index(drop=True)

test_df = model_manifest_df[
    model_manifest_df['dev_split'] == 'test'
].reset_index(drop=True)

train_nonempty_df = train_df[
    train_df['preprocessed_target_nonzero_voxels'] > 0
].reset_index(drop=True)

print(f'Total manifest subjects: {len(model_manifest_df)}')
print(f'Train subjects: {len(train_df)}')
print(f'Non-empty train subjects for Trial2: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')
print('Excluded empty train subjects:', sorted(set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])))


In [ ]:
class ISLESPreprocessedPatchDataset(Dataset):
    def __init__(
        self,
        df,
        patch_size=(128, 128, 64),
        samples_per_epoch=None,
        positive_prob=0.85,
        seed=20240824,
    ):
        self.df = df.reset_index(drop=True)
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch or len(self.df)
        self.positive_prob = positive_prob
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.samples_per_epoch

    def _load_case(self, index):
        row = self.df.iloc[index % len(self.df)]
        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)
        return row['subject_id'], image, target

    def _pad_to_patch(self, image, target):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            pad_spec = (
                (0, 0),
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )
            image = np.pad(image, pad_spec, mode='constant', constant_values=0)
            target = np.pad(target, pad_spec, mode='constant', constant_values=0)

        return image, target

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            center = coords[self.rng.integers(0, len(coords))]
            ch, cw, cd = center
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        subject_id, image, target = self._load_case(index)
        image, target = self._pad_to_patch(image, target)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'subject_id': subject_id,
        }


In [ ]:
for name in [
    'GENERATED_DIRS',
    'train_nonempty_df',
    'ISLESPreprocessedPatchDataset',
    'TRAINING_RUN_NAME',
]:
    print(f'{name}:', name in globals())

print('TRAINING_RUN_NAME:', globals().get('TRAINING_RUN_NAME'))


In [ ]:
from monai.networks.nets import SegResNet
from monai.losses import DiceFocalLoss

PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1024
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.85

MAX_EPOCHS = 80
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
VAL_EVERY = 1


VAL_THRESHOLD = 0.30

EARLY_STOPPING_PATIENCE = 15
MIN_DELTA = 1e-4

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

train_dataset = ISLESPreprocessedPatchDataset(
    train_nonempty_df,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = SegResNet(
    spatial_dims=3,
    in_channels=6,
    out_channels=1,
    init_filters=16,
    blocks_down=(1, 2, 2, 4),
    blocks_up=(1, 1, 1),
    dropout_prob=0.10,
    norm='INSTANCE',
).to(device)

loss_fn = DiceFocalLoss(
    sigmoid=True,
    squared_pred=True,
    lambda_dice=1.0,
    lambda_focal=1.0,
    gamma=2.0,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)

scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'model': 'MONAI SegResNet CNN-only, imbalance-tuned',
    'trial_note': 'Trial2 tests stronger CNN backbone after Trial1 modestly improved over Trial0.',
    'in_channels': 6,
    'out_channels': 1,
    'init_filters': 16,
    'blocks_down': [1, 2, 2, 4],
    'blocks_up': [1, 1, 1],
    'dropout_prob': 0.10,
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'train_subjects': len(train_nonempty_df),
    'excluded_empty_train_subjects': sorted(
        set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])
    ),
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'validation_threshold': VAL_THRESHOLD,
    'loss': 'DiceFocalLoss(sigmoid=True, gamma=2.0)',
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR',
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'seed': SEED,
    'trial1_best_global_threshold_dice_nonempty': 0.248988,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial2 from: {latest_checkpoint_path}')
else:
    print('Starting new Trial2 SegResNet run.')

batch = next(iter(train_loader))
print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))

print(f'Model trainable parameters: {count_parameters(model):,}')
print(f'Validation threshold: {VAL_THRESHOLD}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
import numpy as np
import pandas as pd
import torch
from monai.inferers import sliding_window_inference

VAL_PATCH_SIZE = PATCH_SIZE
VAL_SW_BATCH_SIZE = 1

def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    return image, target

def pad_to_roi_size(image, target, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        pad_spec = (
            (0, 0),
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )
        image = np.pad(image, pad_spec, mode='constant', constant_values=0)
        target = np.pad(target, pad_spec, mode='constant', constant_values=0)

    return image, target

@torch.no_grad()
def validate_full_volume(model, val_df, max_subjects=None):
    model.eval()

    dice_rows = []
    eval_df = val_df if max_subjects is None else val_df.head(max_subjects)

    for eval_idx, (_, row) in enumerate(eval_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        image, target = pad_to_roi_size(image, target, VAL_PATCH_SIZE)

        image_t = torch.from_numpy(image[None]).to(device)

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=VAL_PATCH_SIZE,
            sw_batch_size=VAL_SW_BATCH_SIZE,
            predictor=model,
            overlap=0.5,
        )

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred = prob >= VAL_THRESHOLD

        target_np = target[0] > 0
        dice = dice_score_binary(pred, target_np)

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())

        dice_rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
        })

        print(
            f'  Val {eval_idx}/{len(eval_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}'
        )

        del image_t, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    dice_df = pd.DataFrame(dice_rows)

    mean_dice_all = float(dice_df['dice'].mean()) if len(dice_df) else 0.0
    nonempty_df = dice_df[~dice_df['target_empty']]
    mean_dice_nonempty = float(nonempty_df['dice'].mean()) if len(nonempty_df) else 0.0

    summary = {
        'mean_dice_all': mean_dice_all,
        'mean_dice_nonempty': mean_dice_nonempty,
        'num_cases': int(len(dice_df)),
        'num_empty_targets': int(dice_df['target_empty'].sum()) if len(dice_df) else 0,
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()) if len(dice_df) else 0.0,
    }

    return summary, dice_df


In [ ]:
import time
import numpy as np
import pandas as pd
import torch

def save_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch(epoch):
    model.train()

    epoch_losses = []
    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type='cuda',
            enabled=torch.cuda.is_available(),
        ):
            logits = model(images)
            loss = loss_fn(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = float(loss.detach().cpu())
        epoch_losses.append(loss_value)

        if step == 1 or step % 50 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | '
                f'Step {step:04d}/{len(train_loader)} | '
                f'Loss {loss_value:.4f}'
            )

    mean_loss = float(np.mean(epoch_losses))
    elapsed = time.time() - start_time

    return mean_loss, elapsed

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'{TRAINING_RUN_NAME} Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_loss, elapsed = train_one_epoch(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning validation...')
        val_summary, val_df_epoch = validate_full_volume(model, val_df)

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'lr': current_lr,
        'elapsed_seconds': elapsed,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Trial2 epoch {epoch} complete.')
    print(f'Train loss: {train_loss:.4f}')
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'LR: {current_lr:.8f}')
    print(f'Elapsed: {elapsed / 60:.2f} min')
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after '
            f'{epochs_without_improvement} epochs without validation improvement.'
        )
        break

print('Trial2 training complete.')


In [ ]:
import numpy as np
import pandas as pd
import torch

best_checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()

print(f"Loaded Trial2 best checkpoint epoch: {best_checkpoint['epoch']}")
print(f"Stored best val Dice: {best_checkpoint['best_val_dice']:.4f}")

THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)

rows = []

with torch.no_grad():
    for eval_idx, (_, row) in enumerate(val_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        image, target = pad_to_roi_size(image, target, VAL_PATCH_SIZE)

        image_t = torch.from_numpy(image[None]).to(device)

        logits = sliding_window_inference(
            inputs=image_t,
            roi_size=VAL_PATCH_SIZE,
            sw_batch_size=VAL_SW_BATCH_SIZE,
            predictor=model,
            overlap=0.5,
        )

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        target_np = target[0] > 0
        target_voxels = int(target_np.sum())

        for threshold in THRESHOLDS:
            pred = prob >= threshold

            rows.append({
                'subject_id': sid,
                'threshold': float(threshold),
                'dice': dice_score_binary(pred, target_np),
                'target_voxels': target_voxels,
                'pred_voxels': int(pred.sum()),
                'target_empty': target_voxels == 0,
                'pred_empty': int(pred.sum()) == 0,
            })

        print(f'Completed {eval_idx}/{len(val_df)}: {sid}')

        del image_t, logits
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

trial2_sweep_df = pd.DataFrame(rows)

summary_rows = []
for threshold, group_df in trial2_sweep_df.groupby('threshold'):
    nonempty_df = group_df[~group_df['target_empty']]

    summary_rows.append({
        'threshold': threshold,
        'mean_dice_all': group_df['dice'].mean(),
        'mean_dice_nonempty': nonempty_df['dice'].mean(),
        'num_empty_targets': int(group_df['target_empty'].sum()),
        'mean_pred_voxels': group_df['pred_voxels'].mean(),
        'median_pred_voxels': group_df['pred_voxels'].median(),
        'num_zero_predictions': int(group_df['pred_empty'].sum()),
    })

trial2_threshold_summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values('mean_dice_nonempty', ascending=False)
    .reset_index(drop=True)
)

trial2_case_best_df = (
    trial2_sweep_df
    .sort_values(['subject_id', 'dice'], ascending=[True, False])
    .groupby('subject_id')
    .head(1)
    .reset_index(drop=True)
)

trial2_case_best_with_bins_df = trial2_case_best_df.merge(
    val_df[['subject_id', 'lesion_volume_bin']],
    on='subject_id',
    how='left',
).sort_values('dice', ascending=False)

sweep_detail_path = log_dir / 'trial2_best_checkpoint_threshold_sweep_cases.csv'
sweep_summary_path = log_dir / 'trial2_best_checkpoint_threshold_sweep_summary.csv'
case_best_path = log_dir / 'trial2_best_checkpoint_per_case_best_threshold.csv'

for path in [sweep_detail_path, sweep_summary_path, case_best_path]:
    assert_generated_path_is_safe(path)

trial2_sweep_df.to_csv(sweep_detail_path, index=False)
trial2_threshold_summary_df.to_csv(sweep_summary_path, index=False)
trial2_case_best_with_bins_df.to_csv(case_best_path, index=False)

print('\nBest Trial2 global thresholds:')
display(trial2_threshold_summary_df.head(10))

print('\nTrial2 best per-case Dice:')
display(trial2_case_best_with_bins_df)

print('\nTrial2 summary by lesion size:')
display(
    trial2_case_best_with_bins_df
    .groupby('lesion_volume_bin')
    .agg(
        cases=('subject_id', 'count'),
        mean_best_dice=('dice', 'mean'),
        median_best_dice=('dice', 'median'),
        mean_target_voxels=('target_voxels', 'mean'),
        mean_pred_voxels=('pred_voxels', 'mean'),
    )
    .reset_index()
    .sort_values('mean_best_dice', ascending=False)
)

print(f'Saved threshold sweep: {sweep_summary_path}')
print(f'Saved per-case best summary: {case_best_path}')


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

print('Manifest columns:')
display(pd.DataFrame({'column': model_manifest_df.columns}))

print('\nExample NPZ keys and shapes:')
example_row = train_df.iloc[0]
example_npz_path = example_row['preprocessed_npz_path']

arr = np.load(example_npz_path, allow_pickle=True)

for key in arr.files:
    value = arr[key]
    print(f'{key}: shape={value.shape}, dtype={value.dtype}')

print('\nExample subject:', example_row['subject_id'])
print('Example NPZ:', example_npz_path)

possible_region_terms = [
    'region', 'territory', 'vascular', 'venous', 'artery',
    'atlas', 'mask', 'label', 'seg', 'prior'
]

print('\nColumns that may contain region/prior information:')
for col in model_manifest_df.columns:
    if any(term in col.lower() for term in possible_region_terms):
        print(col)


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from monai.inferers import sliding_window_inference
from monai.losses import DiceFocalLoss

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()
    if path != project_root and project_root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'
TRAINING_RUN_NAME = 'trial3_acfnet_plus_coarse_region_graph_v1'

SEED = 20240824
seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')

MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

train_df = model_manifest_df[model_manifest_df['dev_split'] == 'train'].reset_index(drop=True)
val_df = model_manifest_df[model_manifest_df['dev_split'] == 'val'].reset_index(drop=True)
test_df = model_manifest_df[model_manifest_df['dev_split'] == 'test'].reset_index(drop=True)

train_nonempty_df = train_df[
    train_df['preprocessed_target_nonzero_voxels'] > 0
].reset_index(drop=True)

print(f'Total manifest subjects: {len(model_manifest_df)}')
print(f'Train subjects: {len(train_df)}')
print(f'Non-empty train subjects for Trial3: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')
print('Excluded empty train subjects:', sorted(set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])))


In [ ]:
NUM_REGIONS = 6
REGION_NAMES = [
    'left_anterior',
    'left_middle',
    'left_posterior',
    'right_anterior',
    'right_middle',
    'right_posterior',
]



REGION_EDGES = [
    (0, 1), (1, 2),
    (3, 4), (4, 5),
    (0, 3), (1, 4), (2, 5),
]

def build_region_map(shape_3d):
    h, w, d = shape_3d
    region = np.zeros((h, w, d), dtype=np.int64)

    mid_x = h // 2
    third_y_1 = w // 3
    third_y_2 = 2 * w // 3

    
    region[:mid_x, :third_y_1, :] = 0
    region[:mid_x, third_y_1:third_y_2, :] = 1
    region[:mid_x, third_y_2:, :] = 2

    
    region[mid_x:, :third_y_1, :] = 3
    region[mid_x:, third_y_1:third_y_2, :] = 4
    region[mid_x:, third_y_2:, :] = 5

    return region

def build_adjacency(num_regions=NUM_REGIONS, edges=REGION_EDGES, self_loops=True):
    adj = torch.zeros((num_regions, num_regions), dtype=torch.float32)

    for i, j in edges:
        adj[i, j] = 1.0
        adj[j, i] = 1.0

    if self_loops:
        adj += torch.eye(num_regions, dtype=torch.float32)

    degree = adj.sum(dim=1, keepdim=True).clamp_min(1.0)
    adj = adj / degree
    return adj

example_npz = np.load(train_nonempty_df.iloc[0]['preprocessed_npz_path'], allow_pickle=True)
example_image = example_npz['image']
example_region = build_region_map(example_image.shape[1:])
adjacency = build_adjacency()

print('Example image shape:', example_image.shape)
print('Region map shape:', example_region.shape)
print('Unique regions:', np.unique(example_region))
print('Region names:', REGION_NAMES)
print('Adjacency matrix:')
print(adjacency)


In [ ]:
PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1024
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.85

MAX_EPOCHS = 80
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-5
VAL_EVERY = 1


VAL_THRESHOLD = 0.20

EARLY_STOPPING_PATIENCE = 15
MIN_DELTA = 1e-4
AUX_LOSS_WEIGHT = 0.25

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

class ISLESPreprocessedGraphPatchDataset(Dataset):
    def __init__(self, df, patch_size, samples_per_epoch, positive_prob, seed):
        self.df = df.reset_index(drop=True)
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch
        self.positive_prob = positive_prob
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.samples_per_epoch

    def _load_case(self, index):
        row = self.df.iloc[index % len(self.df)]
        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)
        region = build_region_map(image.shape[1:])
        return row['subject_id'], image, target, region

    def _pad_to_patch(self, image, target, region):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            pad_3d = (
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )
            pad_4d = ((0, 0),) + pad_3d

            image = np.pad(image, pad_4d, mode='constant', constant_values=0)
            target = np.pad(target, pad_4d, mode='constant', constant_values=0)
            region = np.pad(region, pad_3d, mode='edge')

        return image, target, region

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            center = coords[self.rng.integers(0, len(coords))]
            ch, cw, cd = center
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        subject_id, image, target, region = self._load_case(index)
        image, target, region = self._pad_to_patch(image, target, region)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'region': torch.from_numpy(region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]).long(),
            'subject_id': subject_id,
        }

class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()

        self.conv1 = nn.Conv3d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(out_channels)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()

        if in_channels != out_channels or stride != 1:
            self.skip = nn.Conv3d(in_channels, out_channels, 1, stride=stride, bias=False)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        x = self.act(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.act(x + residual)

class GraphConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.linear_self = nn.Linear(channels, channels)
        self.linear_neigh = nn.Linear(channels, channels)
        self.norm = nn.LayerNorm(channels)

    def forward(self, nodes, adjacency):
        neigh = torch.einsum('ij,bjc->bic', adjacency, nodes)
        out = self.linear_self(nodes) + self.linear_neigh(neigh)
        return F.relu(self.norm(out))

class ACFNetPlusCoarseGraph(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, base_channels=16, num_regions=6, graph_layers=2, dropout=0.10):
        super().__init__()

        self.num_regions = num_regions

        self.enc0 = ResidualBlock3D(in_channels, base_channels)
        self.enc1 = ResidualBlock3D(base_channels, base_channels * 2, stride=2)
        self.enc2 = ResidualBlock3D(base_channels * 2, base_channels * 4, stride=2)
        self.enc3 = ResidualBlock3D(base_channels * 4, base_channels * 8, stride=2, dropout=dropout)

        bottleneck_channels = base_channels * 8

        self.graph_layers = nn.ModuleList([
            GraphConv(bottleneck_channels) for _ in range(graph_layers)
        ])

        self.graph_fuse = nn.Sequential(
            nn.Conv3d(bottleneck_channels * 2, bottleneck_channels, 1, bias=False),
            nn.InstanceNorm3d(bottleneck_channels),
            nn.LeakyReLU(0.1, inplace=True),
        )

        self.up2 = nn.ConvTranspose3d(bottleneck_channels, base_channels * 4, 2, stride=2)
        self.dec2 = ResidualBlock3D(base_channels * 8, base_channels * 4)

        self.up1 = nn.ConvTranspose3d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec1 = ResidualBlock3D(base_channels * 4, base_channels * 2)

        self.up0 = nn.ConvTranspose3d(base_channels * 2, base_channels, 2, stride=2)
        self.dec0 = ResidualBlock3D(base_channels * 2, base_channels)

        self.context_head = nn.Conv3d(base_channels, out_channels, 1)

        self.boundary_head = nn.Sequential(
            nn.Conv3d(base_channels, base_channels, 3, padding=1, bias=False),
            nn.InstanceNorm3d(base_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(base_channels, out_channels, 1),
        )

        self.region_head = nn.Sequential(
            nn.Conv3d(base_channels + num_regions, base_channels, 3, padding=1, bias=False),
            nn.InstanceNorm3d(base_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv3d(base_channels, out_channels, 1),
        )

        
        self.router = nn.Sequential(
            nn.Linear(num_regions + 4, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 3),
        )

    def pool_region_nodes(self, features, region_down):
        nodes = []

        for r in range(self.num_regions):
            mask = (region_down == r).float().unsqueeze(1)
            denom = mask.sum(dim=(2, 3, 4)).clamp_min(1.0)
            pooled = (features * mask).sum(dim=(2, 3, 4)) / denom
            nodes.append(pooled)

        return torch.stack(nodes, dim=1)

    def expand_region_nodes(self, nodes, region_down):
        b, nr, c = nodes.shape
        _, h, w, d = region_down.shape

        expanded = torch.zeros((b, c, h, w, d), device=nodes.device, dtype=nodes.dtype)

        for r in range(self.num_regions):
            mask = (region_down == r).unsqueeze(1)
            expanded = expanded + nodes[:, r, :, None, None, None] * mask

        return expanded

    def compute_router_weights(self, logits_stack, region_full):
        b, experts, _, h, w, d = logits_stack.shape
        probs = torch.sigmoid(logits_stack)
        rows = []

        for r in range(self.num_regions):
            mask = (region_full == r).float().unsqueeze(1).unsqueeze(1)
            denom = mask.sum(dim=(3, 4, 5)).clamp_min(1.0)

            confidence = (torch.abs(probs - 0.5) * mask).sum(dim=(3, 4, 5)) / denom
            confidence = confidence.squeeze(2)

            region_fraction = mask.mean(dim=(3, 4, 5)).view(b, 1)

            region_onehot = F.one_hot(
                torch.full((b,), r, device=region_full.device, dtype=torch.long),
                num_classes=self.num_regions,
            ).float()

            descriptor = torch.cat([region_onehot, confidence, region_fraction], dim=1)
            weights = torch.softmax(self.router(descriptor), dim=1)
            rows.append(weights)

        return torch.stack(rows, dim=1)

    def forward(self, image, region):
        region = region.long()

        x0 = self.enc0(image)
        x1 = self.enc1(x0)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)

        region_down = F.interpolate(
            region.unsqueeze(1).float(),
            size=x3.shape[2:],
            mode='nearest',
        ).squeeze(1).long()

        adjacency = build_adjacency().to(image.device)

        nodes = self.pool_region_nodes(x3, region_down)
        for layer in self.graph_layers:
            nodes = layer(nodes, adjacency)

        graph_context = self.expand_region_nodes(nodes, region_down)
        x3 = self.graph_fuse(torch.cat([x3, graph_context], dim=1))

        y2 = self.up2(x3)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))

        y1 = self.up1(y2)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))

        y0 = self.up0(y1)
        y0 = self.dec0(torch.cat([y0, x0], dim=1))

        region_onehot = F.one_hot(
            region.clamp(0, self.num_regions - 1),
            num_classes=self.num_regions,
        ).permute(0, 4, 1, 2, 3).float()

        z_context = self.context_head(y0)
        z_boundary = self.boundary_head(y0)
        z_region = self.region_head(torch.cat([y0, region_onehot], dim=1))

        logits_stack = torch.stack([z_context, z_boundary, z_region], dim=1)
        router_weights = self.compute_router_weights(logits_stack, region)

        b = image.shape[0]
        fused = torch.zeros_like(z_context)

        for r in range(self.num_regions):
            mask = (region == r).float().unsqueeze(1)

            w_context = router_weights[:, r, 0].view(b, 1, 1, 1, 1)
            w_boundary = router_weights[:, r, 1].view(b, 1, 1, 1, 1)
            w_region = router_weights[:, r, 2].view(b, 1, 1, 1, 1)

            fused_region = (
                w_context * z_context
                + w_boundary * z_boundary
                + w_region * z_region
            )

            fused = fused + fused_region * mask

        return {
            'logits': fused,
            'context_logits': z_context,
            'boundary_logits': z_boundary,
            'region_logits': z_region,
            'router_weights': router_weights,
        }

train_dataset = ISLESPreprocessedGraphPatchDataset(
    train_nonempty_df,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = ACFNetPlusCoarseGraph(
    in_channels=6,
    out_channels=1,
    base_channels=16,
    num_regions=NUM_REGIONS,
    graph_layers=2,
    dropout=0.10,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Trial3 ACF-Net+ total parameters: {total_params:,}')
print(f'Trial3 ACF-Net+ trainable parameters: {trainable_params:,}')

loss_fn = DiceFocalLoss(
    sigmoid=True,
    squared_pred=True,
    lambda_dice=1.0,
    lambda_focal=1.0,
    gamma=2.0,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)

scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'model': 'ACF-Net+ coarse regional graph prototype',
    'trial_note': 'Trial3 adds region pooling, graph conditioning, expert heads, and learned region-wise fusion.',
    'region_prior': 'six coarse image-space regions: left/right x anterior/middle/posterior',
    'region_names': REGION_NAMES,
    'region_edges': REGION_EDGES,
    'in_channels': 6,
    'out_channels': 1,
    'base_channels': 16,
    'num_regions': NUM_REGIONS,
    'graph_layers': 2,
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'aux_loss_weight': AUX_LOSS_WEIGHT,
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'validation_threshold': VAL_THRESHOLD,
    'loss': 'fused DiceFocalLoss + auxiliary expert DiceFocalLoss',
    'optimizer': 'AdamW',
    'scheduler': 'CosineAnnealingLR',
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'seed': SEED,
    'trial2_best_global_threshold_dice_nonempty': 0.260500,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial3 from: {latest_checkpoint_path}')
else:
    print('Starting new Trial3 ACF-Net+ run.')

batch = next(iter(train_loader))
images = batch['image'].to(device)
targets = batch['target'].to(device)
regions = batch['region'].to(device)

with torch.no_grad():
    outputs = model(images, regions)

print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch region:', batch['region'].shape, batch['region'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))
print('Output logits:', outputs['logits'].shape)
print('Router weights:', outputs['router_weights'].shape)
print('Mean router weights:', outputs['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print(f'Validation threshold: {VAL_THRESHOLD}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
import numpy as np
import pandas as pd
import torch

VAL_PATCH_SIZE = PATCH_SIZE
VAL_SW_BATCH_SIZE = 1

def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    region = build_region_map(image.shape[1:])
    return image, target, region

def pad_full_case(image, target, region, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        pad_3d = (
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )
        pad_4d = ((0, 0),) + pad_3d

        image = np.pad(image, pad_4d, mode='constant', constant_values=0)
        target = np.pad(target, pad_4d, mode='constant', constant_values=0)
        region = np.pad(region, pad_3d, mode='edge')

    return image, target, region

def get_window_starts(size, roi):
    if size <= roi:
        return [0]

    starts = list(range(0, size - roi + 1, roi // 2))

    if starts[-1] != size - roi:
        starts.append(size - roi)

    return starts

@torch.no_grad()
def predict_full_volume_graph(model, image, region, roi_size):
    model.eval()

    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    logits_sum = torch.zeros((1, 1, h, w, d), device=device)
    count_map = torch.zeros((1, 1, h, w, d), device=device)

    image_t = torch.from_numpy(image[None]).to(device)
    region_t = torch.from_numpy(region[None]).long().to(device)

    h_starts = get_window_starts(h, ph)
    w_starts = get_window_starts(w, pw)
    d_starts = get_window_starts(d, pd_)

    router_values = []

    for sh in h_starts:
        for sw in w_starts:
            for sd in d_starts:
                image_patch = image_t[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_]
                region_patch = region_t[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]

                outputs = model(image_patch, region_patch)
                logits_patch = outputs['logits']

                logits_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += logits_patch
                count_map[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += 1.0

                router_values.append(outputs['router_weights'].detach().cpu())

    logits = logits_sum / count_map.clamp_min(1.0)
    prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

    router_mean = torch.cat(router_values, dim=0).mean(dim=(0, 1)).numpy()

    del image_t, region_t, logits_sum, count_map, logits
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return prob, router_mean

@torch.no_grad()
def validate_full_volume(model, val_df, max_subjects=None):
    model.eval()

    dice_rows = []
    router_rows = []
    eval_df = val_df if max_subjects is None else val_df.head(max_subjects)

    for eval_idx, (_, row) in enumerate(eval_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target, region = load_full_case_npz(row['preprocessed_npz_path'])
        image, target, region = pad_full_case(image, target, region, VAL_PATCH_SIZE)

        prob, router_mean = predict_full_volume_graph(model, image, region, VAL_PATCH_SIZE)

        pred = prob >= VAL_THRESHOLD
        target_np = target[0] > 0

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())
        dice = dice_score_binary(pred, target_np)

        dice_rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
        })

        router_rows.append({
            'subject_id': sid,
            'context_weight': float(router_mean[0]),
            'boundary_weight': float(router_mean[1]),
            'region_weight': float(router_mean[2]),
        })

        print(
            f'  Val {eval_idx}/{len(eval_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}, '
            f'router={router_mean}'
        )

    dice_df = pd.DataFrame(dice_rows)
    router_df = pd.DataFrame(router_rows)

    mean_dice_all = float(dice_df['dice'].mean()) if len(dice_df) else 0.0
    nonempty_df = dice_df[~dice_df['target_empty']]
    mean_dice_nonempty = float(nonempty_df['dice'].mean()) if len(nonempty_df) else 0.0

    summary = {
        'mean_dice_all': mean_dice_all,
        'mean_dice_nonempty': mean_dice_nonempty,
        'num_cases': int(len(dice_df)),
        'num_empty_targets': int(dice_df['target_empty'].sum()) if len(dice_df) else 0,
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()) if len(dice_df) else 0.0,
        'mean_context_weight': float(router_df['context_weight'].mean()) if len(router_df) else 0.0,
        'mean_boundary_weight': float(router_df['boundary_weight'].mean()) if len(router_df) else 0.0,
        'mean_region_weight': float(router_df['region_weight'].mean()) if len(router_df) else 0.0,
    }

    return summary, dice_df, router_df


In [ ]:
import time
import numpy as np
import pandas as pd
import torch

def compute_trial3_loss(outputs, targets):
    fused_loss = loss_fn(outputs['logits'], targets)

    aux_loss = (
        loss_fn(outputs['context_logits'], targets)
        + loss_fn(outputs['boundary_logits'], targets)
        + loss_fn(outputs['region_logits'], targets)
    ) / 3.0

    return fused_loss + AUX_LOSS_WEIGHT * aux_loss

def save_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch(epoch):
    model.train()
    epoch_losses = []
    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        regions = batch['region'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type='cuda',
            enabled=torch.cuda.is_available(),
        ):
            outputs = model(images, regions)
            loss = compute_trial3_loss(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = float(loss.detach().cpu())
        epoch_losses.append(loss_value)

        if step == 1 or step % 50 == 0 or step == len(train_loader):
            router_mean = outputs['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy()
            print(
                f'Epoch {epoch:03d} | '
                f'Step {step:04d}/{len(train_loader)} | '
                f'Loss {loss_value:.4f} | '
                f'Router {router_mean}'
            )

    mean_loss = float(np.mean(epoch_losses))
    elapsed = time.time() - start_time

    return mean_loss, elapsed

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'{TRAINING_RUN_NAME} Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_loss, elapsed = train_one_epoch(epoch)

    val_summary = None
    val_df_epoch = None
    router_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning validation...')
        val_summary, val_df_epoch, router_df_epoch = validate_full_volume(model, val_df)

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        router_detail_path = log_dir / f'router_epoch_{epoch:03d}.csv'

        assert_generated_path_is_safe(val_detail_path)
        assert_generated_path_is_safe(router_detail_path)

        val_df_epoch.to_csv(val_detail_path, index=False)
        router_df_epoch.to_csv(router_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(
            "Validation mean router weights: "
            f"context={val_summary['mean_context_weight']:.4f}, "
            f"boundary={val_summary['mean_boundary_weight']:.4f}, "
            f"region={val_summary['mean_region_weight']:.4f}"
        )

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'mean_context_weight': None if val_summary is None else val_summary['mean_context_weight'],
        'mean_boundary_weight': None if val_summary is None else val_summary['mean_boundary_weight'],
        'mean_region_weight': None if val_summary is None else val_summary['mean_region_weight'],
        'lr': current_lr,
        'elapsed_seconds': elapsed,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Trial3 epoch {epoch} complete.')
    print(f'Train loss: {train_loss:.4f}')
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'LR: {current_lr:.8f}')
    print(f'Elapsed: {elapsed / 60:.2f} min')
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after '
            f'{epochs_without_improvement} epochs without validation improvement.'
        )
        break

print('Trial3 training complete.')


In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

TRIAL2_BEST_SWEEPED_NONEMPTY_DICE = 0.260500

TRIAL3_SWEEP_THRESHOLDS = [
    0.05, 0.10, 0.15, 0.20, 0.25,
    0.30, 0.35, 0.40, 0.45, 0.50,
    0.55, 0.60, 0.65, 0.70, 0.75, 0.80
]

best_checkpoint_path = checkpoint_dir / 'best.pt'
assert best_checkpoint_path.exists(), f'Missing best checkpoint: {best_checkpoint_path}'

checkpoint = torch.load(best_checkpoint_path, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"Loaded Trial3 best checkpoint epoch: {checkpoint['epoch']}")
print(f"Stored best val Dice: {checkpoint.get('best_val_dice', None):.4f}")

def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    return image, target

def pad_image_target_region_to_roi(image, target, region, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        spatial_pad = (
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )

        image = np.pad(
            image,
            ((0, 0),) + spatial_pad,
            mode='constant',
            constant_values=0,
        )
        target = np.pad(
            target,
            ((0, 0),) + spatial_pad,
            mode='constant',
            constant_values=0,
        )
        region = np.pad(
            region,
            spatial_pad,
            mode='constant',
            constant_values=0,
        )

    return image, target, region

@torch.no_grad()
def predict_full_volume_graph_prob(model, image, region, roi_size):
    model.eval()

    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    logits_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)
    count_map = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)

    router_rows = []

    h_starts = list(range(0, max(h - ph, 0) + 1, ph // 2))
    w_starts = list(range(0, max(w - pw, 0) + 1, pw // 2))
    d_starts = list(range(0, max(d - pd_, 0) + 1, pd_ // 2))

    if h_starts[-1] != h - ph:
        h_starts.append(h - ph)
    if w_starts[-1] != w - pw:
        w_starts.append(w - pw)
    if d_starts[-1] != d - pd_:
        d_starts.append(d - pd_)

    image_t = torch.from_numpy(image).to(device)
    region_t = torch.from_numpy(region).long().to(device)

    for sh in h_starts:
        for sw in w_starts:
            for sd in d_starts:
                image_patch = image_t[
                    None,
                    :,
                    sh:sh+ph,
                    sw:sw+pw,
                    sd:sd+pd_,
                ]
                region_patch = region_t[
                    None,
                    sh:sh+ph,
                    sw:sw+pw,
                    sd:sd+pd_,
                ]

                outputs = model(image_patch, region_patch)
                logits_patch = outputs['logits']

                logits_sum[
                    :,
                    :,
                    sh:sh+ph,
                    sw:sw+pw,
                    sd:sd+pd_,
                ] += logits_patch

                count_map[
                    :,
                    :,
                    sh:sh+ph,
                    sw:sw+pw,
                    sd:sd+pd_,
                ] += 1

                router_rows.append(
                    outputs['router_weights']
                    .detach()
                    .mean(dim=(0, 1))
                    .cpu()
                    .numpy()
                )

    logits = logits_sum / torch.clamp(count_map, min=1.0)
    prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()

    mean_router = np.vstack(router_rows).mean(axis=0)

    return prob, mean_router

case_rows = []

for case_idx, (_, row) in enumerate(val_df.iterrows(), start=1):
    sid = row['subject_id']

    image, target = load_full_case_npz(row['preprocessed_npz_path'])
    region = build_region_map(image.shape[1:])

    image, target, region = pad_image_target_region_to_roi(
        image=image,
        target=target,
        region=region,
        roi_size=PATCH_SIZE,
    )

    prob, mean_router = predict_full_volume_graph_prob(
        model=model,
        image=image,
        region=region,
        roi_size=PATCH_SIZE,
    )

    target_np = target[0] > 0
    target_voxels = int(target_np.sum())

    for threshold in TRIAL3_SWEEP_THRESHOLDS:
        pred = prob >= threshold
        pred_voxels = int(pred.sum())

        case_rows.append({
            'subject_id': sid,
            'threshold': threshold,
            'dice': dice_score_binary(pred, target_np),
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
            'lesion_volume_bin': row['lesion_volume_bin'],
            'router_context': float(mean_router[0]),
            'router_boundary': float(mean_router[1]),
            'router_region': float(mean_router[2]),
        })

    print(f'Completed {case_idx}/{len(val_df)}: {sid}')

sweep_cases_df = pd.DataFrame(case_rows)

threshold_summary_df = (
    sweep_cases_df
    .groupby('threshold', as_index=False)
    .agg(
        mean_dice_all=('dice', 'mean'),
        mean_dice_nonempty=(
            'dice',
            lambda s: s[
                ~sweep_cases_df.loc[s.index, 'target_empty']
            ].mean()
        ),
        num_empty_targets=('target_empty', 'sum'),
        mean_pred_voxels=('pred_voxels', 'mean'),
        median_pred_voxels=('pred_voxels', 'median'),
        num_zero_predictions=('pred_empty', 'sum'),
        mean_router_context=('router_context', 'mean'),
        mean_router_boundary=('router_boundary', 'mean'),
        mean_router_region=('router_region', 'mean'),
    )
    .sort_values('mean_dice_nonempty', ascending=False)
    .reset_index(drop=True)
)

best_threshold = float(threshold_summary_df.iloc[0]['threshold'])
best_sweep_dice = float(threshold_summary_df.iloc[0]['mean_dice_nonempty'])

print('\nBest Trial3 global thresholds:')
display(threshold_summary_df)

print('\nBest Trial3 threshold result:')
print(f'Best threshold: {best_threshold:.2f}')
print(f'Best non-empty val Dice: {best_sweep_dice:.4f}')
print(f'Trial2 best swept non-empty val Dice: {TRIAL2_BEST_SWEEPED_NONEMPTY_DICE:.4f}')
print(f'Delta vs Trial2: {best_sweep_dice - TRIAL2_BEST_SWEEPED_NONEMPTY_DICE:+.4f}')

sweep_cases_path = log_dir / 'trial3_best_checkpoint_threshold_sweep_cases.csv'
threshold_summary_path = log_dir / 'trial3_best_checkpoint_threshold_sweep_summary.csv'

assert_generated_path_is_safe(sweep_cases_path)
assert_generated_path_is_safe(threshold_summary_path)

sweep_cases_df.to_csv(sweep_cases_path, index=False)
threshold_summary_df.to_csv(threshold_summary_path, index=False)

print(f'\nSaved case-level sweep: {sweep_cases_path}')
print(f'Saved threshold summary: {threshold_summary_path}')


In [ ]:
per_case_best_df = (
    sweep_cases_df
    .sort_values(['subject_id', 'dice'], ascending=[True, False])
    .groupby('subject_id', as_index=False)
    .head(1)
    .sort_values('dice', ascending=False)
    .reset_index(drop=True)
)

size_summary_df = (
    per_case_best_df
    .groupby('lesion_volume_bin', as_index=False)
    .agg(
        cases=('subject_id', 'count'),
        mean_best_dice=('dice', 'mean'),
        median_best_dice=('dice', 'median'),
        mean_target_voxels=('target_voxels', 'mean'),
        mean_pred_voxels=('pred_voxels', 'mean'),
        mean_router_context=('router_context', 'mean'),
        mean_router_boundary=('router_boundary', 'mean'),
        mean_router_region=('router_region', 'mean'),
    )
    .sort_values('mean_best_dice', ascending=False)
)

per_case_best_path = log_dir / 'trial3_best_checkpoint_per_case_best_threshold.csv'
assert_generated_path_is_safe(per_case_best_path)
per_case_best_df.to_csv(per_case_best_path, index=False)

print('Trial3 best per-case validation Dice after threshold sweep:')
display(per_case_best_df)

print('\nTrial3 summary by lesion size:')
display(size_summary_df)

print(f'\nSaved Trial3 per-case best summary: {per_case_best_path}')


In [ ]:
import os
import json
import time
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from google.colab import drive

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()
    if project_root not in path.parents and path != project_root:
        raise ValueError(f'Unsafe path outside project root: {path}')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'
TRAINING_RUN_NAME = 'trial4_acfnet_plus_upgraded_v1'

SEED = 20240824
PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1024
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.90

MAX_EPOCHS = 90
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 2e-5
VAL_EVERY = 1
VAL_THRESHOLD = 0.15
EARLY_STOPPING_PATIENCE = 18
MIN_DELTA = 1e-4

BASE_CHANNELS = 24
DROPOUT = 0.15
AUX_LOSS_WEIGHT = 0.10
ROUTER_ENTROPY_WEIGHT = 0.01
EXPERT_DROPOUT_PROB = 0.15

TRIAL2_BEST_SWEEPED_NONEMPTY_DICE = 0.260500
TRIAL3_BEST_SWEEPED_NONEMPTY_DICE = 0.257794

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')

MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

train_df = model_manifest_df[model_manifest_df['dev_split'] == 'train'].reset_index(drop=True)
val_df = model_manifest_df[model_manifest_df['dev_split'] == 'val'].reset_index(drop=True)
test_df = model_manifest_df[model_manifest_df['dev_split'] == 'test'].reset_index(drop=True)

train_nonempty_df = train_df[
    train_df['preprocessed_target_nonzero_voxels'] > 0
].reset_index(drop=True)

print(f'Total manifest subjects: {len(model_manifest_df)}')
print(f'Train subjects: {len(train_df)}')
print(f'Non-empty train subjects for Trial4: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')
print(
    'Excluded empty train subjects:',
    sorted(set(train_df['subject_id']) - set(train_nonempty_df['subject_id']))
)

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

NUM_REGIONS = 6
REGION_NAMES = [
    'left_anterior',
    'left_middle',
    'left_posterior',
    'right_anterior',
    'right_middle',
    'right_posterior',
]

REGION_EDGES = [
    (0, 1), (1, 2),
    (3, 4), (4, 5),
    (0, 3), (1, 4), (2, 5),
]

def build_region_map(shape_3d):
    h, w, d = shape_3d
    region = np.zeros((h, w, d), dtype=np.int64)

    h_mid = h // 2
    w1 = w // 3
    w2 = 2 * w // 3

    region[:h_mid, :w1, :] = 0
    region[:h_mid, w1:w2, :] = 1
    region[:h_mid, w2:, :] = 2
    region[h_mid:, :w1, :] = 3
    region[h_mid:, w1:w2, :] = 4
    region[h_mid:, w2:, :] = 5

    return region

def build_adjacency(num_regions=NUM_REGIONS, edges=REGION_EDGES):
    adj = torch.eye(num_regions, dtype=torch.float32)

    for src, dst in edges:
        adj[src, dst] = 1.0
        adj[dst, src] = 1.0

    adj = adj / adj.sum(dim=1, keepdim=True).clamp_min(1.0)
    return adj

ADJACENCY = build_adjacency()

class ISLESBalancedGraphPatchDataset(Dataset):
    def __init__(
        self,
        df,
        patch_size=(128, 128, 64),
        samples_per_epoch=1024,
        positive_prob=0.90,
        seed=20240824,
    ):
        self.df = df.reset_index(drop=True)
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch
        self.positive_prob = positive_prob
        self.rng = np.random.default_rng(seed)

        self.bin_to_indices = {
            lesion_bin: group.index.to_numpy()
            for lesion_bin, group in self.df.groupby('lesion_volume_bin')
        }
        self.available_bins = sorted(self.bin_to_indices.keys())

    def __len__(self):
        return self.samples_per_epoch

    def _sample_case_index(self):
        lesion_bin = self.rng.choice(self.available_bins)
        return int(self.rng.choice(self.bin_to_indices[lesion_bin]))

    def _load_case(self):
        row = self.df.iloc[self._sample_case_index()]
        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)
        region = build_region_map(image.shape[1:])
        return row['subject_id'], image, target, region

    def _pad_to_patch(self, image, target, region):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            spatial_pad = (
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )

            image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant')
            target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant')
            region = np.pad(region, spatial_pad, mode='constant')

        return image, target, region

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            ch, cw, cd = coords[self.rng.integers(0, len(coords))]
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        subject_id, image, target, region = self._load_case()
        image, target, region = self._pad_to_patch(image, target, region)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'region': torch.from_numpy(region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]).long(),
            'subject_id': subject_id,
        }

class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
        )

        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        return self.act(self.block(x) + self.shortcut(x))

class GraphConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.linear = nn.Linear(channels, channels)
        self.norm = nn.LayerNorm(channels)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, node_features, adjacency):
        mixed = torch.einsum('ij,bjc->bic', adjacency, node_features)
        return self.act(self.norm(self.linear(mixed) + node_features))

class ACFNetPlusCoarseGraph(nn.Module):
    def __init__(
        self,
        in_channels=6,
        out_channels=1,
        base_channels=24,
        num_regions=6,
        adjacency=None,
        graph_layers=2,
        dropout=0.15,
        expert_dropout_prob=0.15,
    ):
        super().__init__()

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8

        self.num_regions = num_regions
        self.expert_dropout_prob = expert_dropout_prob

        self.enc0 = ResidualBlock3D(in_channels, c1, dropout=dropout)
        self.down1 = nn.Conv3d(c1, c2, kernel_size=3, stride=2, padding=1)
        self.enc1 = ResidualBlock3D(c2, c2, dropout=dropout)
        self.down2 = nn.Conv3d(c2, c3, kernel_size=3, stride=2, padding=1)
        self.enc2 = ResidualBlock3D(c3, c3, dropout=dropout)
        self.down3 = nn.Conv3d(c3, c4, kernel_size=3, stride=2, padding=1)
        self.enc3 = ResidualBlock3D(c4, c4, dropout=dropout)

        self.graph_layers = nn.ModuleList([GraphConv(c4) for _ in range(graph_layers)])
        self.graph_project = nn.Sequential(
            nn.Linear(c4, c4),
            nn.LayerNorm(c4),
            nn.LeakyReLU(0.1, inplace=True),
        )

        self.up2 = nn.ConvTranspose3d(c4, c3, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock3D(c3 + c3, c3, dropout=dropout)
        self.up1 = nn.ConvTranspose3d(c3, c2, kernel_size=2, stride=2)
        self.dec1 = ResidualBlock3D(c2 + c2, c2, dropout=dropout)
        self.up0 = nn.ConvTranspose3d(c2, c1, kernel_size=2, stride=2)
        self.dec0 = ResidualBlock3D(c1 + c1, c1, dropout=dropout)

        self.context_head = nn.Conv3d(c1, out_channels, kernel_size=1)
        self.boundary_head = nn.Conv3d(c1, out_channels, kernel_size=1)
        self.region_head = nn.Conv3d(c1, out_channels, kernel_size=1)

        router_in = num_regions + 4
        self.router = nn.Sequential(
            nn.Linear(router_in, 48),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(48, 3),
        )

        if adjacency is None:
            adjacency = torch.eye(num_regions, dtype=torch.float32)

        self.register_buffer('adjacency', adjacency.float())

    def _region_one_hot(self, region, spatial_size):
        region_resized = F.interpolate(
            region[:, None].float(),
            size=spatial_size,
            mode='nearest',
        ).long().squeeze(1)

        region_resized = region_resized.clamp(0, self.num_regions - 1)

        one_hot = F.one_hot(
            region_resized,
            num_classes=self.num_regions,
        ).permute(0, 4, 1, 2, 3).float()

        return one_hot

    def _pool_regions(self, features, one_hot):
        denom = one_hot.sum(dim=(2, 3, 4)).clamp_min(1.0)
        pooled = torch.einsum('bchwd,brhwd->brc', features, one_hot)
        return pooled / denom[:, :, None]

    def _router_weights(self, region, expert_logits):
        full_one_hot = self._region_one_hot(region, expert_logits[0].shape[2:])
        region_fraction = full_one_hot.mean(dim=(2, 3, 4), keepdim=False)[:, :, None]

        expert_conf = []
        for logits in expert_logits:
            prob = torch.sigmoid(logits)
            pooled_prob = self._pool_regions(prob, full_one_hot)
            expert_conf.append(pooled_prob[:, :, 0])

        expert_conf = torch.stack(expert_conf, dim=-1)

        identity = torch.eye(
            self.num_regions,
            device=region.device,
            dtype=torch.float32,
        )[None].repeat(region.shape[0], 1, 1)

        router_input = torch.cat(
            [identity, expert_conf, region_fraction],
            dim=-1,
        )

        router_logits = self.router(router_input)

        if self.training and self.expert_dropout_prob > 0:
            drop_mask = torch.rand_like(router_logits) < self.expert_dropout_prob
            all_dropped = drop_mask.all(dim=-1, keepdim=True)
            drop_mask = drop_mask & ~all_dropped
            router_logits = router_logits.masked_fill(drop_mask, -1e4)

        router_weights = torch.softmax(router_logits, dim=-1)
        return router_weights, full_one_hot

    def forward(self, image, region):
        x0 = self.enc0(image)
        x1 = self.enc1(self.down1(x0))
        x2 = self.enc2(self.down2(x1))
        x3 = self.enc3(self.down3(x2))

        bottleneck_one_hot = self._region_one_hot(region, x3.shape[2:])
        node_features = self._pool_regions(x3, bottleneck_one_hot)

        for layer in self.graph_layers:
            node_features = layer(node_features, self.adjacency)

        node_features = self.graph_project(node_features)
        graph_volume = torch.einsum('brc,brhwd->bchwd', node_features, bottleneck_one_hot)
        x3 = x3 + graph_volume

        y2 = self.up2(x3)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))

        y1 = self.up1(y2)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))

        y0 = self.up0(y1)
        y0 = self.dec0(torch.cat([y0, x0], dim=1))

        context_logits = self.context_head(y0)
        boundary_logits = self.boundary_head(y0)
        region_logits = self.region_head(y0)

        router_weights, full_one_hot = self._router_weights(
            region,
            [context_logits, boundary_logits, region_logits],
        )

        expert_maps = torch.stack(
            [
                context_logits[:, 0],
                boundary_logits[:, 0],
                region_logits[:, 0],
            ],
            dim=1,
        )

        weight_maps = torch.einsum('bre,brhwd->behwd', router_weights, full_one_hot)
        logits = (expert_maps * weight_maps).sum(dim=1, keepdim=True)

        return {
            'logits': logits,
            'context_logits': context_logits,
            'boundary_logits': boundary_logits,
            'region_logits': region_logits,
            'router_weights': router_weights,
        }

class DiceFocalTverskyLoss(nn.Module):
    def __init__(
        self,
        alpha=0.25,
        beta=0.75,
        gamma=1.33,
        lambda_dice=1.0,
        lambda_tversky=1.0,
        lambda_bce=0.25,
        smooth=1e-5,
    ):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lambda_dice = lambda_dice
        self.lambda_tversky = lambda_tversky
        self.lambda_bce = lambda_bce
        self.smooth = smooth

    def forward(self, logits, target):
        prob = torch.sigmoid(logits)
        target = target.float()

        dims = (2, 3, 4)

        tp = (prob * target).sum(dim=dims)
        fp = (prob * (1.0 - target)).sum(dim=dims)
        fn = ((1.0 - prob) * target).sum(dim=dims)

        dice = (2.0 * tp + self.smooth) / (
            2.0 * tp + fp + fn + self.smooth
        )
        dice_loss = 1.0 - dice.mean()

        tversky = (tp + self.smooth) / (
            tp + self.alpha * fp + self.beta * fn + self.smooth
        )
        focal_tversky_loss = torch.pow(1.0 - tversky, self.gamma).mean()

        bce_loss = F.binary_cross_entropy_with_logits(logits, target)

        return (
            self.lambda_dice * dice_loss
            + self.lambda_tversky * focal_tversky_loss
            + self.lambda_bce * bce_loss
        )

train_dataset = ISLESBalancedGraphPatchDataset(
    train_nonempty_df,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = ACFNetPlusCoarseGraph(
    in_channels=6,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    num_regions=NUM_REGIONS,
    adjacency=ADJACENCY,
    graph_layers=2,
    dropout=DROPOUT,
    expert_dropout_prob=EXPERT_DROPOUT_PROB,
).to(device)

loss_fn = DiceFocalTverskyLoss(
    alpha=0.25,
    beta=0.75,
    gamma=1.33,
    lambda_dice=1.0,
    lambda_tversky=1.0,
    lambda_bce=0.25,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)

try:
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'trial_note': 'Performance-seeking ACF-Net+ Trial4: larger capacity, size-balanced sampling, focal Tversky, router entropy, expert dropout.',
    'model': 'ACF-Net+ coarse 6-region graph upgraded',
    'base_channels': BASE_CHANNELS,
    'dropout': DROPOUT,
    'expert_dropout_prob': EXPERT_DROPOUT_PROB,
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'aux_loss_weight': AUX_LOSS_WEIGHT,
    'router_entropy_weight': ROUTER_ENTROPY_WEIGHT,
    'loss': 'DiceFocalTverskyLoss(alpha=0.25,beta=0.75,gamma=1.33)',
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'validation_threshold': VAL_THRESHOLD,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'trial2_best_sweeped_nonempty_dice': TRIAL2_BEST_SWEEPED_NONEMPTY_DICE,
    'trial3_best_sweeped_nonempty_dice': TRIAL3_BEST_SWEEPED_NONEMPTY_DICE,
    'seed': SEED,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial4 from: {latest_checkpoint_path}')
    print(f'Start epoch: {start_epoch}')
    print(f'Best val Dice so far: {best_val_dice:.4f}')
    print(f'Epochs without improvement: {epochs_without_improvement}')
else:
    print('Starting new Trial4 ACF-Net+ upgraded run.')

batch = next(iter(train_loader))
images = batch['image'].to(device)
targets = batch['target'].to(device)
regions = batch['region'].to(device)

model.eval()
with torch.no_grad():
    outputs = model(images, regions)

print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch region:', batch['region'].shape, batch['region'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))
print('Output logits:', outputs['logits'].shape)
print('Router weights:', outputs['router_weights'].shape)
print('Mean router weights:', outputs['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print(f'Trial4 ACF-Net+ trainable parameters: {count_parameters(model):,}')
print(f'Validation threshold: {VAL_THRESHOLD}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    return image, target

def pad_image_target_region_to_roi(image, target, region, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        spatial_pad = (
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )

        image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant')
        target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant')
        region = np.pad(region, spatial_pad, mode='constant')

    return image, target, region

@torch.no_grad()
def predict_full_volume_graph_prob(model, image, region, roi_size):
    model.eval()

    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    logits_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)
    count_map = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)

    router_rows = []

    h_starts = list(range(0, max(h - ph, 0) + 1, ph // 2))
    w_starts = list(range(0, max(w - pw, 0) + 1, pw // 2))
    d_starts = list(range(0, max(d - pd_, 0) + 1, pd_ // 2))

    if h_starts[-1] != h - ph:
        h_starts.append(h - ph)
    if w_starts[-1] != w - pw:
        w_starts.append(w - pw)
    if d_starts[-1] != d - pd_:
        d_starts.append(d - pd_)

    image_t = torch.from_numpy(image).to(device)
    region_t = torch.from_numpy(region).long().to(device)

    for sh in h_starts:
        for sw in w_starts:
            for sd in d_starts:
                image_patch = image_t[None, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_]
                region_patch = region_t[None, sh:sh+ph, sw:sw+pw, sd:sd+pd_]

                outputs = model(image_patch, region_patch)
                logits_patch = outputs['logits']

                logits_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += logits_patch
                count_map[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += 1

                router_rows.append(
                    outputs['router_weights']
                    .detach()
                    .mean(dim=(0, 1))
                    .cpu()
                    .numpy()
                )

    logits = logits_sum / torch.clamp(count_map, min=1.0)
    prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
    mean_router = np.vstack(router_rows).mean(axis=0)

    return prob, mean_router

@torch.no_grad()
def validate_full_volume_trial4(model, val_df):
    model.eval()

    dice_rows = []

    for case_idx, (_, row) in enumerate(val_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        region = build_region_map(image.shape[1:])

        image, target, region = pad_image_target_region_to_roi(
            image=image,
            target=target,
            region=region,
            roi_size=PATCH_SIZE,
        )

        prob, mean_router = predict_full_volume_graph_prob(
            model=model,
            image=image,
            region=region,
            roi_size=PATCH_SIZE,
        )

        pred = prob >= VAL_THRESHOLD
        target_np = target[0] > 0

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())
        dice = dice_score_binary(pred, target_np)

        dice_rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
            'lesion_volume_bin': row['lesion_volume_bin'],
            'router_context': float(mean_router[0]),
            'router_boundary': float(mean_router[1]),
            'router_region': float(mean_router[2]),
        })

        print(
            f'  Val {case_idx}/{len(val_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}, '
            f'router={mean_router}'
        )

    dice_df = pd.DataFrame(dice_rows)
    nonempty_df = dice_df[~dice_df['target_empty']]

    summary = {
        'mean_dice_all': float(dice_df['dice'].mean()),
        'mean_dice_nonempty': float(nonempty_df['dice'].mean()),
        'num_empty_targets': int(dice_df['target_empty'].sum()),
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()),
        'mean_router_context': float(dice_df['router_context'].mean()),
        'mean_router_boundary': float(dice_df['router_boundary'].mean()),
        'mean_router_region': float(dice_df['router_region'].mean()),
    }

    return summary, dice_df

def router_entropy_loss(router_weights, eps=1e-8):
    entropy = -(router_weights * torch.log(router_weights + eps)).sum(dim=-1)
    normalized_entropy = entropy / math.log(router_weights.shape[-1])
    return -normalized_entropy.mean()

def compute_trial4_loss(outputs, targets):
    main_loss = loss_fn(outputs['logits'], targets)

    aux_loss = (
        loss_fn(outputs['context_logits'], targets)
        + loss_fn(outputs['boundary_logits'], targets)
        + loss_fn(outputs['region_logits'], targets)
    ) / 3.0

    entropy_loss = router_entropy_loss(outputs['router_weights'])

    total_loss = (
        main_loss
        + AUX_LOSS_WEIGHT * aux_loss
        + ROUTER_ENTROPY_WEIGHT * entropy_loss
    )

    return total_loss, {
        'main_loss': float(main_loss.detach().cpu()),
        'aux_loss': float(aux_loss.detach().cpu()),
        'router_entropy_loss': float(entropy_loss.detach().cpu()),
    }

def save_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch_trial4(epoch):
    model.train()

    losses = []
    main_losses = []
    aux_losses = []
    entropy_losses = []
    router_rows = []

    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        regions = batch['region'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        try:
            autocast_context = torch.amp.autocast('cuda', enabled=torch.cuda.is_available())
        except TypeError:
            autocast_context = torch.cuda.amp.autocast(enabled=torch.cuda.is_available())

        with autocast_context:
            outputs = model(images, regions)
            loss, parts = compute_trial4_loss(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.detach().cpu()))
        main_losses.append(parts['main_loss'])
        aux_losses.append(parts['aux_loss'])
        entropy_losses.append(parts['router_entropy_loss'])

        mean_router = outputs['router_weights'].detach().mean(dim=(0, 1)).cpu().numpy()
        router_rows.append(mean_router)

        if step == 1 or step % 50 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | Step {step:04d}/{len(train_loader)} | '
                f'Loss {losses[-1]:.4f} | Main {main_losses[-1]:.4f} | '
                f'Aux {aux_losses[-1]:.4f} | Router {mean_router}'
            )

    elapsed = time.time() - start_time
    router_mean = np.vstack(router_rows).mean(axis=0)

    return {
        'train_loss': float(np.mean(losses)),
        'train_main_loss': float(np.mean(main_losses)),
        'train_aux_loss': float(np.mean(aux_losses)),
        'train_router_entropy_loss': float(np.mean(entropy_losses)),
        'train_router_context': float(router_mean[0]),
        'train_router_boundary': float(router_mean[1]),
        'train_router_region': float(router_mean[2]),
        'elapsed_seconds': elapsed,
    }

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'{TRAINING_RUN_NAME} Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_summary = train_one_epoch_trial4(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning Trial4 validation...')
        val_summary, val_df_epoch = validate_full_volume_trial4(model, val_df)

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(
            'Validation mean router weights: '
            f"context={val_summary['mean_router_context']:.4f}, "
            f"boundary={val_summary['mean_router_boundary']:.4f}, "
            f"region={val_summary['mean_router_region']:.4f}"
        )
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_row = {
        'epoch': epoch,
        **train_summary,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'val_router_context': None if val_summary is None else val_summary['mean_router_context'],
        'val_router_boundary': None if val_summary is None else val_summary['mean_router_boundary'],
        'val_router_region': None if val_summary is None else val_summary['mean_router_region'],
        'lr': current_lr,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Trial4 epoch {epoch} complete.')
    print(f"Train loss: {train_summary['train_loss']:.4f}")
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'LR: {current_lr:.8f}')
    print(f"Elapsed: {train_summary['elapsed_seconds'] / 60:.2f} min")
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after '
            f'{epochs_without_improvement} epochs without validation improvement.'
        )
        break

print('Trial4 training complete.')


In [ ]:
import os
import json
import time
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from google.colab import drive
drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()
    if project_root not in path.parents and path != project_root:
        raise ValueError(f'Unsafe path outside project root: {path}')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def require_finite_tensor(name, tensor):
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f'{name} contains NaN or Inf.')

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'
TRAINING_RUN_NAME = 'trial4_acfnet_plus_upgraded_v2'

SEED = 20240824
PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1024
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.90

MAX_EPOCHS = 90
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 2e-5
VAL_EVERY = 1
VAL_THRESHOLD = 0.15
EARLY_STOPPING_PATIENCE = 18
MIN_DELTA = 1e-4

BASE_CHANNELS = 24
DROPOUT = 0.15
AUX_LOSS_WEIGHT = 0.10
ROUTER_ENTROPY_WEIGHT = 0.001
EXPERT_DROPOUT_PROB = 0.0
GRAD_CLIP_NORM = 1.0

TRIAL2_BEST_SWEEPED_NONEMPTY_DICE = 0.260500
TRIAL3_BEST_SWEEPED_NONEMPTY_DICE = 0.257794

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')

MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

train_df = model_manifest_df[model_manifest_df['dev_split'] == 'train'].reset_index(drop=True)
val_df = model_manifest_df[model_manifest_df['dev_split'] == 'val'].reset_index(drop=True)
test_df = model_manifest_df[model_manifest_df['dev_split'] == 'test'].reset_index(drop=True)

train_nonempty_df = train_df[
    train_df['preprocessed_target_nonzero_voxels'] > 0
].reset_index(drop=True)

print(f'Total manifest subjects: {len(model_manifest_df)}')
print(f'Train subjects: {len(train_df)}')
print(f'Non-empty train subjects for Trial4 v2: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')
print('Excluded empty train subjects:', sorted(set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])))

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

NUM_REGIONS = 6
REGION_NAMES = [
    'left_anterior', 'left_middle', 'left_posterior',
    'right_anterior', 'right_middle', 'right_posterior',
]

REGION_EDGES = [
    (0, 1), (1, 2),
    (3, 4), (4, 5),
    (0, 3), (1, 4), (2, 5),
]

def build_region_map(shape_3d):
    h, w, d = shape_3d
    region = np.zeros((h, w, d), dtype=np.int64)

    h_mid = h // 2
    w1 = w // 3
    w2 = 2 * w // 3

    region[:h_mid, :w1, :] = 0
    region[:h_mid, w1:w2, :] = 1
    region[:h_mid, w2:, :] = 2
    region[h_mid:, :w1, :] = 3
    region[h_mid:, w1:w2, :] = 4
    region[h_mid:, w2:, :] = 5

    return region

def build_adjacency(num_regions=NUM_REGIONS, edges=REGION_EDGES):
    adj = torch.eye(num_regions, dtype=torch.float32)

    for src, dst in edges:
        adj[src, dst] = 1.0
        adj[dst, src] = 1.0

    return adj / adj.sum(dim=1, keepdim=True).clamp_min(1.0)

ADJACENCY = build_adjacency()

class ISLESBalancedGraphPatchDataset(Dataset):
    def __init__(self, df, patch_size, samples_per_epoch, positive_prob, seed):
        self.df = df.reset_index(drop=True)
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch
        self.positive_prob = positive_prob
        self.rng = np.random.default_rng(seed)

        self.bin_to_indices = {
            lesion_bin: group.index.to_numpy()
            for lesion_bin, group in self.df.groupby('lesion_volume_bin')
        }
        self.available_bins = sorted(self.bin_to_indices.keys())

    def __len__(self):
        return self.samples_per_epoch

    def _sample_case_index(self):
        lesion_bin = self.rng.choice(self.available_bins)
        return int(self.rng.choice(self.bin_to_indices[lesion_bin]))

    def _load_case(self):
        row = self.df.iloc[self._sample_case_index()]
        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)

        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)
        region = build_region_map(image.shape[1:])

        return row['subject_id'], image, target, region

    def _pad_to_patch(self, image, target, region):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            spatial_pad = (
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )

            image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant')
            target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant')
            region = np.pad(region, spatial_pad, mode='constant')

        return image, target, region

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = self.rng.random() < self.positive_prob and np.count_nonzero(target) > 0

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            ch, cw, cd = coords[self.rng.integers(0, len(coords))]
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        subject_id, image, target, region = self._load_case()
        image, target, region = self._pad_to_patch(image, target, region)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'region': torch.from_numpy(region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]).long(),
            'subject_id': subject_id,
        }

class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
        )

        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        return self.act(self.block(x) + self.shortcut(x))

class GraphConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.linear = nn.Linear(channels, channels)
        self.norm = nn.LayerNorm(channels)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, node_features, adjacency):
        mixed = torch.einsum('ij,bjc->bic', adjacency, node_features)
        return self.act(self.norm(self.linear(mixed) + node_features))

class ACFNetPlusCoarseGraph(nn.Module):
    def __init__(
        self,
        in_channels=6,
        out_channels=1,
        base_channels=24,
        num_regions=6,
        adjacency=None,
        graph_layers=2,
        dropout=0.15,
        expert_dropout_prob=0.0,
    ):
        super().__init__()

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8

        self.num_regions = num_regions
        self.expert_dropout_prob = expert_dropout_prob

        self.enc0 = ResidualBlock3D(in_channels, c1, dropout=dropout)
        self.down1 = nn.Conv3d(c1, c2, kernel_size=3, stride=2, padding=1)
        self.enc1 = ResidualBlock3D(c2, c2, dropout=dropout)
        self.down2 = nn.Conv3d(c2, c3, kernel_size=3, stride=2, padding=1)
        self.enc2 = ResidualBlock3D(c3, c3, dropout=dropout)
        self.down3 = nn.Conv3d(c3, c4, kernel_size=3, stride=2, padding=1)
        self.enc3 = ResidualBlock3D(c4, c4, dropout=dropout)

        self.graph_layers = nn.ModuleList([GraphConv(c4) for _ in range(graph_layers)])
        self.graph_project = nn.Sequential(
            nn.Linear(c4, c4),
            nn.LayerNorm(c4),
            nn.LeakyReLU(0.1, inplace=True),
        )

        self.up2 = nn.ConvTranspose3d(c4, c3, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock3D(c3 + c3, c3, dropout=dropout)
        self.up1 = nn.ConvTranspose3d(c3, c2, kernel_size=2, stride=2)
        self.dec1 = ResidualBlock3D(c2 + c2, c2, dropout=dropout)
        self.up0 = nn.ConvTranspose3d(c2, c1, kernel_size=2, stride=2)
        self.dec0 = ResidualBlock3D(c1 + c1, c1, dropout=dropout)

        self.context_head = nn.Conv3d(c1, out_channels, kernel_size=1)
        self.boundary_head = nn.Conv3d(c1, out_channels, kernel_size=1)
        self.region_head = nn.Conv3d(c1, out_channels, kernel_size=1)

        self.router = nn.Sequential(
            nn.Linear(num_regions + 4, 48),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(48, 3),
        )

        if adjacency is None:
            adjacency = torch.eye(num_regions, dtype=torch.float32)

        self.register_buffer('adjacency', adjacency.float())

    def _region_one_hot(self, region, spatial_size):
        region_resized = F.interpolate(
            region[:, None].float(),
            size=spatial_size,
            mode='nearest',
        ).long().squeeze(1)

        region_resized = region_resized.clamp(0, self.num_regions - 1)

        return F.one_hot(
            region_resized,
            num_classes=self.num_regions,
        ).permute(0, 4, 1, 2, 3).float()

    def _pool_regions(self, features, one_hot):
        denom = one_hot.sum(dim=(2, 3, 4)).clamp_min(1.0)
        pooled = torch.einsum('bchwd,brhwd->brc', features, one_hot)
        return pooled / denom[:, :, None]

    def _router_weights(self, region, expert_logits):
        full_one_hot = self._region_one_hot(region, expert_logits[0].shape[2:])
        region_fraction = full_one_hot.mean(dim=(2, 3, 4))[:, :, None]

        expert_conf = []
        for logits in expert_logits:
            prob = torch.sigmoid(logits.float())
            pooled_prob = self._pool_regions(prob, full_one_hot)
            expert_conf.append(pooled_prob[:, :, 0])

        expert_conf = torch.stack(expert_conf, dim=-1)

        identity = torch.eye(
            self.num_regions,
            device=region.device,
            dtype=torch.float32,
        )[None].repeat(region.shape[0], 1, 1)

        router_input = torch.cat([identity, expert_conf, region_fraction], dim=-1)
        router_logits = self.router(router_input.float())

        router_weights = torch.softmax(router_logits.float(), dim=-1)
        return router_weights, full_one_hot

    def forward(self, image, region):
        x0 = self.enc0(image.float())
        x1 = self.enc1(self.down1(x0))
        x2 = self.enc2(self.down2(x1))
        x3 = self.enc3(self.down3(x2))

        bottleneck_one_hot = self._region_one_hot(region, x3.shape[2:])
        node_features = self._pool_regions(x3, bottleneck_one_hot)

        for layer in self.graph_layers:
            node_features = layer(node_features, self.adjacency)

        node_features = self.graph_project(node_features)
        graph_volume = torch.einsum('brc,brhwd->bchwd', node_features, bottleneck_one_hot)
        x3 = x3 + graph_volume

        y2 = self.up2(x3)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))

        y1 = self.up1(y2)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))

        y0 = self.up0(y1)
        y0 = self.dec0(torch.cat([y0, x0], dim=1))

        context_logits = self.context_head(y0)
        boundary_logits = self.boundary_head(y0)
        region_logits = self.region_head(y0)

        router_weights, full_one_hot = self._router_weights(
            region,
            [context_logits, boundary_logits, region_logits],
        )

        expert_maps = torch.stack(
            [context_logits[:, 0], boundary_logits[:, 0], region_logits[:, 0]],
            dim=1,
        )

        weight_maps = torch.einsum('bre,brhwd->behwd', router_weights, full_one_hot)
        logits = (expert_maps * weight_maps).sum(dim=1, keepdim=True)

        return {
            'logits': logits,
            'context_logits': context_logits,
            'boundary_logits': boundary_logits,
            'region_logits': region_logits,
            'router_weights': router_weights,
        }

class DiceFocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.25, beta=0.75, gamma=1.33, lambda_dice=1.0, lambda_tversky=1.0, lambda_bce=0.25, smooth=1e-5):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lambda_dice = lambda_dice
        self.lambda_tversky = lambda_tversky
        self.lambda_bce = lambda_bce
        self.smooth = smooth

    def forward(self, logits, target):
        require_finite_tensor('loss logits', logits)

        logits = logits.float()
        target = target.float()

        prob = torch.sigmoid(logits).clamp(1e-6, 1.0 - 1e-6)
        dims = tuple(range(2, prob.ndim))

        tp = (prob * target).sum(dim=dims)
        fp = (prob * (1.0 - target)).sum(dim=dims)
        fn = ((1.0 - prob) * target).sum(dim=dims)

        dice = (2.0 * tp + self.smooth) / (2.0 * tp + fp + fn + self.smooth)
        dice_loss = 1.0 - dice.clamp(0.0, 1.0).mean()

        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        focal_tversky_loss = torch.pow(
            (1.0 - tversky.clamp(0.0, 1.0)).clamp_min(1e-6),
            self.gamma,
        ).mean()

        bce_loss = F.binary_cross_entropy_with_logits(logits, target)

        total = (
            self.lambda_dice * dice_loss
            + self.lambda_tversky * focal_tversky_loss
            + self.lambda_bce * bce_loss
        )

        require_finite_tensor('total loss', total)
        return total

train_dataset = ISLESBalancedGraphPatchDataset(
    train_nonempty_df,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = ACFNetPlusCoarseGraph(
    in_channels=6,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    num_regions=NUM_REGIONS,
    adjacency=ADJACENCY,
    graph_layers=2,
    dropout=DROPOUT,
    expert_dropout_prob=EXPERT_DROPOUT_PROB,
).to(device)

loss_fn = DiceFocalTverskyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'trial_note': 'Trial4 v2 stabilized after Trial4 v1 immediate NaNs.',
    'stability_note': 'No AMP, no expert dropout, float32 loss, clamped router entropy, gradient clipping, immediate NaN guard.',
    'model': 'ACF-Net+ coarse 6-region graph upgraded stabilized',
    'base_channels': BASE_CHANNELS,
    'dropout': DROPOUT,
    'expert_dropout_prob': EXPERT_DROPOUT_PROB,
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'aux_loss_weight': AUX_LOSS_WEIGHT,
    'router_entropy_weight': ROUTER_ENTROPY_WEIGHT,
    'grad_clip_norm': GRAD_CLIP_NORM,
    'loss': 'DiceFocalTverskyLoss full float32',
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'validation_threshold': VAL_THRESHOLD,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'trial2_best_sweeped_nonempty_dice': TRIAL2_BEST_SWEEPED_NONEMPTY_DICE,
    'trial3_best_sweeped_nonempty_dice': TRIAL3_BEST_SWEEPED_NONEMPTY_DICE,
    'seed': SEED,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial4 v2 from: {latest_checkpoint_path}')
    print(f'Start epoch: {start_epoch}')
    print(f'Best val Dice so far: {best_val_dice:.4f}')
    print(f'Epochs without improvement: {epochs_without_improvement}')
else:
    print('Starting new Trial4 v2 stabilized run.')

batch = next(iter(train_loader))
images = batch['image'].to(device)
targets = batch['target'].to(device)
regions = batch['region'].to(device)

model.eval()
with torch.no_grad():
    outputs = model(images, regions)

require_finite_tensor('smoke logits', outputs['logits'])
require_finite_tensor('smoke router weights', outputs['router_weights'])

print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch region:', batch['region'].shape, batch['region'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))
print('Output logits:', outputs['logits'].shape)
print('Router weights:', outputs['router_weights'].shape)
print('Mean router weights:', outputs['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print(f'Trial4 v2 trainable parameters: {count_parameters(model):,}')
print(f'Validation threshold: {VAL_THRESHOLD}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
model.train()

batch = next(iter(train_loader))
images = batch['image'].to(device)
targets = batch['target'].to(device)
regions = batch['region'].to(device)

optimizer.zero_grad(set_to_none=True)

outputs = model(images, regions)

require_finite_tensor('preflight logits', outputs['logits'])
require_finite_tensor('preflight context logits', outputs['context_logits'])
require_finite_tensor('preflight boundary logits', outputs['boundary_logits'])
require_finite_tensor('preflight region logits', outputs['region_logits'])
require_finite_tensor('preflight router weights', outputs['router_weights'])

def router_entropy_loss(router_weights, eps=1e-4):
    require_finite_tensor('router weights before entropy', router_weights)

    router_weights = router_weights.float().clamp(min=eps, max=1.0)
    router_weights = router_weights / router_weights.sum(dim=-1, keepdim=True).clamp_min(eps)

    entropy = -(router_weights * torch.log(router_weights)).sum(dim=-1)
    normalized_entropy = entropy / math.log(router_weights.shape[-1])

    return -normalized_entropy.mean()

def compute_trial4_loss(outputs, targets):
    targets = targets.float()

    main_loss = loss_fn(outputs['logits'], targets)

    aux_loss = (
        loss_fn(outputs['context_logits'], targets)
        + loss_fn(outputs['boundary_logits'], targets)
        + loss_fn(outputs['region_logits'], targets)
    ) / 3.0

    entropy_loss = router_entropy_loss(outputs['router_weights'])

    total_loss = (
        main_loss
        + AUX_LOSS_WEIGHT * aux_loss
        + ROUTER_ENTROPY_WEIGHT * entropy_loss
    )

    require_finite_tensor('trial4 total loss', total_loss)

    return total_loss, {
        'main_loss': float(main_loss.detach().cpu()),
        'aux_loss': float(aux_loss.detach().cpu()),
        'router_entropy_loss': float(entropy_loss.detach().cpu()),
    }

loss, parts = compute_trial4_loss(outputs, targets)

print('Preflight loss:', float(loss.detach().cpu()))
print('Preflight parts:', parts)
print('Preflight router:', outputs['router_weights'].detach().mean(dim=(0, 1)).cpu().numpy())

loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

print('Preflight grad norm:', float(grad_norm.detach().cpu()))
assert torch.isfinite(grad_norm), 'Preflight failed: gradient norm is NaN or Inf'

optimizer.zero_grad(set_to_none=True)

print('Trial4 v2 preflight passed. Start the training cell next.')


In [ ]:
def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    return image, target

def pad_image_target_region_to_roi(image, target, region, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        spatial_pad = (
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )

        image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant')
        target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant')
        region = np.pad(region, spatial_pad, mode='constant')

    return image, target, region

@torch.no_grad()
def predict_full_volume_graph_prob(model, image, region, roi_size):
    model.eval()

    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    logits_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)
    count_map = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)

    router_rows = []

    h_starts = list(range(0, max(h - ph, 0) + 1, ph // 2))
    w_starts = list(range(0, max(w - pw, 0) + 1, pw // 2))
    d_starts = list(range(0, max(d - pd_, 0) + 1, pd_ // 2))

    if h_starts[-1] != h - ph:
        h_starts.append(h - ph)
    if w_starts[-1] != w - pw:
        w_starts.append(w - pw)
    if d_starts[-1] != d - pd_:
        d_starts.append(d - pd_)

    image_t = torch.from_numpy(image).to(device)
    region_t = torch.from_numpy(region).long().to(device)

    for sh in h_starts:
        for sw in w_starts:
            for sd in d_starts:
                image_patch = image_t[None, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_]
                region_patch = region_t[None, sh:sh+ph, sw:sw+pw, sd:sd+pd_]

                outputs = model(image_patch, region_patch)
                require_finite_tensor('validation logits patch', outputs['logits'])

                logits_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += outputs['logits']
                count_map[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += 1

                router_rows.append(
                    outputs['router_weights']
                    .detach()
                    .mean(dim=(0, 1))
                    .cpu()
                    .numpy()
                )

    logits = logits_sum / torch.clamp(count_map, min=1.0)
    prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
    mean_router = np.vstack(router_rows).mean(axis=0)

    return prob, mean_router

@torch.no_grad()
def validate_full_volume_trial4(model, val_df):
    model.eval()

    dice_rows = []

    for case_idx, (_, row) in enumerate(val_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        region = build_region_map(image.shape[1:])

        image, target, region = pad_image_target_region_to_roi(
            image=image,
            target=target,
            region=region,
            roi_size=PATCH_SIZE,
        )

        prob, mean_router = predict_full_volume_graph_prob(
            model=model,
            image=image,
            region=region,
            roi_size=PATCH_SIZE,
        )

        pred = prob >= VAL_THRESHOLD
        target_np = target[0] > 0

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())
        dice = dice_score_binary(pred, target_np)

        dice_rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
            'lesion_volume_bin': row['lesion_volume_bin'],
            'router_context': float(mean_router[0]),
            'router_boundary': float(mean_router[1]),
            'router_region': float(mean_router[2]),
        })

        print(
            f'  Val {case_idx}/{len(val_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}, '
            f'router={mean_router}'
        )

    dice_df = pd.DataFrame(dice_rows)
    nonempty_df = dice_df[~dice_df['target_empty']]

    summary = {
        'mean_dice_all': float(dice_df['dice'].mean()),
        'mean_dice_nonempty': float(nonempty_df['dice'].mean()),
        'num_empty_targets': int(dice_df['target_empty'].sum()),
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()),
        'mean_router_context': float(dice_df['router_context'].mean()),
        'mean_router_boundary': float(dice_df['router_boundary'].mean()),
        'mean_router_region': float(dice_df['router_region'].mean()),
    }

    return summary, dice_df

def save_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch_trial4(epoch):
    model.train()

    losses = []
    main_losses = []
    aux_losses = []
    entropy_losses = []
    router_rows = []

    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        regions = batch['region'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images, regions)

        require_finite_tensor('train logits', outputs['logits'])
        require_finite_tensor('train router weights', outputs['router_weights'])

        loss, parts = compute_trial4_loss(outputs, targets)

        if not torch.isfinite(loss):
            print('Invalid loss detected.')
            print('Loss parts:', parts)
            print('Batch subject:', batch['subject_id'])
            print('Router:', outputs['router_weights'].detach().mean(dim=(0, 1)).cpu().numpy())
            raise RuntimeError('Stopping Trial4 v2 because loss became NaN or Inf.')

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        if not torch.isfinite(grad_norm):
            raise RuntimeError('Stopping Trial4 v2 because gradient norm became NaN or Inf.')

        optimizer.step()

        losses.append(float(loss.detach().cpu()))
        main_losses.append(parts['main_loss'])
        aux_losses.append(parts['aux_loss'])
        entropy_losses.append(parts['router_entropy_loss'])

        mean_router = outputs['router_weights'].detach().mean(dim=(0, 1)).cpu().numpy()
        router_rows.append(mean_router)

        if step == 1 or step % 50 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | Step {step:04d}/{len(train_loader)} | '
                f'Loss {losses[-1]:.4f} | Main {main_losses[-1]:.4f} | '
                f'Aux {aux_losses[-1]:.4f} | Grad {float(grad_norm):.4f} | '
                f'Router {mean_router}'
            )

    elapsed = time.time() - start_time
    router_mean = np.vstack(router_rows).mean(axis=0)

    return {
        'train_loss': float(np.mean(losses)),
        'train_main_loss': float(np.mean(main_losses)),
        'train_aux_loss': float(np.mean(aux_losses)),
        'train_router_entropy_loss': float(np.mean(entropy_losses)),
        'train_router_context': float(router_mean[0]),
        'train_router_boundary': float(router_mean[1]),
        'train_router_region': float(router_mean[2]),
        'elapsed_seconds': elapsed,
    }

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'{TRAINING_RUN_NAME} Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_summary = train_one_epoch_trial4(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning Trial4 v2 validation...')
        val_summary, val_df_epoch = validate_full_volume_trial4(model, val_df)

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(
            'Validation mean router weights: '
            f"context={val_summary['mean_router_context']:.4f}, "
            f"boundary={val_summary['mean_router_boundary']:.4f}, "
            f"region={val_summary['mean_router_region']:.4f}"
        )
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_row = {
        'epoch': epoch,
        **train_summary,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'val_router_context': None if val_summary is None else val_summary['mean_router_context'],
        'val_router_boundary': None if val_summary is None else val_summary['mean_router_boundary'],
        'val_router_region': None if val_summary is None else val_summary['mean_router_region'],
        'lr': current_lr,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Trial4 v2 epoch {epoch} complete.')
    print(f"Train loss: {train_summary['train_loss']:.4f}")
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'LR: {current_lr:.8f}')
    print(f"Elapsed: {train_summary['elapsed_seconds'] / 60:.2f} min")
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after '
            f'{epochs_without_improvement} epochs without validation improvement.'
        )
        break

print('Trial4 v2 training complete.')


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'logs': PROJECT_ROOT / 'logs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()
    if project_root not in path.parents and path != project_root:
        raise ValueError(f'Unsafe path outside project root: {path}')

PREPROCESSING_VERSION = 'v1_trial0'
REGIONAL_FEATURE_VERSION = 'coarse6_regional_features_v1'

MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

feature_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION / REGIONAL_FEATURE_VERSION
feature_root.mkdir(parents=True, exist_ok=True)
assert_generated_path_is_safe(feature_root)

REGION_NAMES = [
    'left_anterior', 'left_middle', 'left_posterior',
    'right_anterior', 'right_middle', 'right_posterior',
]

CHANNEL_NAMES = ['ncct', 'cta', 'cbf', 'cbv', 'mtt', 'tmax']

def build_region_map(shape_3d):
    h, w, d = shape_3d
    region = np.zeros((h, w, d), dtype=np.int64)

    h_mid = h // 2
    w1 = w // 3
    w2 = 2 * w // 3

    region[:h_mid, :w1, :] = 0
    region[:h_mid, w1:w2, :] = 1
    region[:h_mid, w2:, :] = 2
    region[h_mid:, :w1, :] = 3
    region[h_mid:, w1:w2, :] = 4
    region[h_mid:, w2:, :] = 5

    return region

def safe_percentile(values, q):
    if values.size == 0:
        return 0.0
    return float(np.percentile(values, q))

def extract_regional_features(image):
    region_map = build_region_map(image.shape[1:])
    brain_mask = np.any(image > 0, axis=0)

    feature_names = []
    rows = []

    brain_values = {}
    for c, name in enumerate(CHANNEL_NAMES):
        vals = image[c][brain_mask]
        brain_values[name] = vals[vals > 0] if vals.size else vals

    thresholds = {
        'cbf_low': safe_percentile(brain_values['cbf'], 10),
        'cbv_low': safe_percentile(brain_values['cbv'], 10),
        'mtt_high': safe_percentile(brain_values['mtt'], 90),
        'tmax_high': safe_percentile(brain_values['tmax'], 90),
    }

    for region_id, region_name in enumerate(REGION_NAMES):
        mask = (region_map == region_id) & brain_mask
        if mask.sum() == 0:
            mask = region_map == region_id

        feats = []

        names_this_region = []
        for c, channel_name in enumerate(CHANNEL_NAMES):
            vals = image[c][mask]
            feats.extend([
                float(vals.mean()) if vals.size else 0.0,
                float(vals.std()) if vals.size else 0.0,
                safe_percentile(vals, 10),
                safe_percentile(vals, 50),
                safe_percentile(vals, 90),
            ])
            names_this_region.extend([
                f'{channel_name}_mean',
                f'{channel_name}_std',
                f'{channel_name}_p10',
                f'{channel_name}_p50',
                f'{channel_name}_p90',
            ])

        feats.extend([
            float((image[2][mask] <= thresholds['cbf_low']).mean()) if mask.sum() else 0.0,
            float((image[3][mask] <= thresholds['cbv_low']).mean()) if mask.sum() else 0.0,
            float((image[4][mask] >= thresholds['mtt_high']).mean()) if mask.sum() else 0.0,
            float((image[5][mask] >= thresholds['tmax_high']).mean()) if mask.sum() else 0.0,
        ])
        names_this_region.extend([
            'cbf_low_fraction',
            'cbv_low_fraction',
            'mtt_high_fraction',
            'tmax_high_fraction',
        ])

        coords = np.argwhere(region_map == region_id)
        center = coords.mean(axis=0) / np.array(region_map.shape)

        feats.extend([
            float(mask.mean()),
            float(center[0]),
            float(center[1]),
            float(center[2]),
            float(region_id in [0, 1, 2]),
            float(region_id in [3, 4, 5]),
        ])
        names_this_region.extend([
            'region_brain_fraction',
            'center_h_norm',
            'center_w_norm',
            'center_d_norm',
            'is_left',
            'is_right',
        ])

        if region_id == 0:
            feature_names = names_this_region

        rows.append(feats)

    features = np.asarray(rows, dtype=np.float32)

    
    asym = np.zeros_like(features)
    pairs = [(0, 3), (1, 4), (2, 5)]

    for left_id, right_id in pairs:
        delta = features[left_id] - features[right_id]
        asym[left_id] = delta
        asym[right_id] = -delta

    features = np.concatenate([features, asym], axis=1)
    feature_names = feature_names + [f'asym_{name}' for name in feature_names]

    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    return features, np.asarray(feature_names)

audit_rows = []

for idx, row in model_manifest_df.iterrows():
    sid = row['subject_id']
    npz_path = row['preprocessed_npz_path']
    out_path = feature_root / f'{sid}_{REGIONAL_FEATURE_VERSION}.npz'

    try:
        arr = np.load(npz_path, allow_pickle=True)
        image = arr['image'].astype(np.float32)

        features, feature_names = extract_regional_features(image)

        np.savez_compressed(
            out_path,
            subject_id=sid,
            regional_features=features,
            feature_names=feature_names,
            region_names=np.asarray(REGION_NAMES),
            regional_feature_version=REGIONAL_FEATURE_VERSION,
            preprocessing_version=PREPROCESSING_VERSION,
        )

        audit_rows.append({
            'subject_id': sid,
            'regional_feature_path': str(out_path),
            'readable': True,
            'num_regions': features.shape[0],
            'num_features': features.shape[1],
            'finite': bool(np.isfinite(features).all()),
            'error': None,
        })

    except Exception as exc:
        audit_rows.append({
            'subject_id': sid,
            'regional_feature_path': str(out_path),
            'readable': False,
            'num_regions': None,
            'num_features': None,
            'finite': False,
            'error': repr(exc),
        })

audit_df = pd.DataFrame(audit_rows)
audit_path = feature_root / 'regional_feature_audit.csv'
assert_generated_path_is_safe(audit_path)
audit_df.to_csv(audit_path, index=False)

print(f'Saved regional feature sidecars: {feature_root}')
print(f'Saved audit: {audit_path}')
print(f'Total subjects: {len(audit_df)}')
print(f'Valid sidecars: {int((audit_df["readable"] & audit_df["finite"]).sum())}')
print('\nFeature shape example:')
display(audit_df.head())


In [ ]:
import json
import math
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from monai.losses import DiceFocalLoss
from google.colab import drive

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()
    if project_root not in path.parents and path != project_root:
        raise ValueError(f'Unsafe path outside project root: {path}')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

PREPROCESSING_VERSION = 'v1_trial0'
REGIONAL_FEATURE_VERSION = 'coarse6_regional_features_v1'
TRAINING_RUN_NAME = 'trial5_acfnet_plus_regional_features_v1'

SEED = 20240824
PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1024
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.90

MAX_EPOCHS = 90
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 2e-5
VAL_EVERY = 1
VAL_THRESHOLD = 0.15
EARLY_STOPPING_PATIENCE = 18
MIN_DELTA = 1e-4

BASE_CHANNELS = 24
DROPOUT = 0.15
AUX_LOSS_WEIGHT = 0.10

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')

manifest_path = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
feature_root = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION / REGIONAL_FEATURE_VERSION

model_manifest_df = pd.read_csv(manifest_path)
model_manifest_df['regional_feature_path'] = model_manifest_df['subject_id'].apply(
    lambda sid: str(feature_root / f'{sid}_{REGIONAL_FEATURE_VERSION}.npz')
)

train_df = model_manifest_df[model_manifest_df['dev_split'] == 'train'].reset_index(drop=True)
val_df = model_manifest_df[model_manifest_df['dev_split'] == 'val'].reset_index(drop=True)
test_df = model_manifest_df[model_manifest_df['dev_split'] == 'test'].reset_index(drop=True)

train_nonempty_df = train_df[train_df['preprocessed_target_nonzero_voxels'] > 0].reset_index(drop=True)

def load_regional_features(path):
    arr = np.load(path, allow_pickle=True)
    return arr['regional_features'].astype(np.float32)

train_feature_stack = np.stack([
    load_regional_features(path)
    for path in train_nonempty_df['regional_feature_path']
])

feature_mean = train_feature_stack.mean(axis=(0, 1), keepdims=True).astype(np.float32)
feature_std = train_feature_stack.std(axis=(0, 1), keepdims=True).astype(np.float32)
feature_std = np.where(feature_std < 1e-6, 1.0, feature_std).astype(np.float32)

NODE_FEATURE_DIM = train_feature_stack.shape[-1]
NUM_REGIONS = train_feature_stack.shape[1]

print(f'Total manifest subjects: {len(model_manifest_df)}')
print(f'Non-empty train subjects for Trial5: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')
print(f'Node feature shape: regions={NUM_REGIONS}, features={NODE_FEATURE_DIM}')

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'


In [ ]:
NUM_REGIONS = 6

REGION_NAMES = [
    'left_anterior', 'left_middle', 'left_posterior',
    'right_anterior', 'right_middle', 'right_posterior',
]

REGION_EDGES = [
    (0, 1), (1, 2),
    (3, 4), (4, 5),
    (0, 3), (1, 4), (2, 5),
]

def build_region_map(shape_3d):
    h, w, d = shape_3d
    region = np.zeros((h, w, d), dtype=np.int64)

    h_mid = h // 2
    w1 = w // 3
    w2 = 2 * w // 3

    region[:h_mid, :w1, :] = 0
    region[:h_mid, w1:w2, :] = 1
    region[:h_mid, w2:, :] = 2
    region[h_mid:, :w1, :] = 3
    region[h_mid:, w1:w2, :] = 4
    region[h_mid:, w2:, :] = 5

    return region

def build_adjacency(num_regions=NUM_REGIONS, edges=REGION_EDGES):
    adj = torch.eye(num_regions, dtype=torch.float32)

    for src, dst in edges:
        adj[src, dst] = 1.0
        adj[dst, src] = 1.0

    adj = adj / adj.sum(dim=1, keepdim=True).clamp_min(1.0)
    return adj

ADJACENCY = build_adjacency()

print('Region names:', REGION_NAMES)
print('Adjacency:')
print(ADJACENCY)


In [ ]:
GRAD_CLIP_NORM = 1.0

class ISLESRegionalFeatureGraphPatchDataset(Dataset):
    def __init__(self, df, patch_size, samples_per_epoch, positive_prob, seed):
        self.df = df.reset_index(drop=True)
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch
        self.positive_prob = positive_prob
        self.rng = np.random.default_rng(seed)

        self.bin_to_indices = {
            lesion_bin: group.index.to_numpy()
            for lesion_bin, group in self.df.groupby('lesion_volume_bin')
        }
        self.available_bins = sorted(self.bin_to_indices.keys())

    def __len__(self):
        return self.samples_per_epoch

    def _sample_case_index(self):
        lesion_bin = self.rng.choice(self.available_bins)
        return int(self.rng.choice(self.bin_to_indices[lesion_bin]))

    def _load_case(self):
        row = self.df.iloc[self._sample_case_index()]

        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)
        region = build_region_map(image.shape[1:])

        node_features = load_regional_features(row['regional_feature_path'])
        node_features = (node_features - feature_mean[0]) / feature_std[0]
        node_features = np.nan_to_num(node_features, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

        return row['subject_id'], image, target, region, node_features

    def _pad_to_patch(self, image, target, region):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            spatial_pad = (
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )

            image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant')
            target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant')
            region = np.pad(region, spatial_pad, mode='constant')

        return image, target, region

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            ch, cw, cd = coords[self.rng.integers(0, len(coords))]
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        subject_id, image, target, region, node_features = self._load_case()
        image, target, region = self._pad_to_patch(image, target, region)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'region': torch.from_numpy(region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]).long(),
            'node_features': torch.from_numpy(node_features),
            'subject_id': subject_id,
        }

class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout3d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
        )

        self.shortcut = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        return self.act(self.block(x) + self.shortcut(x))

class GraphConv(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.linear = nn.Linear(channels, channels)
        self.norm = nn.LayerNorm(channels)
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, node_features, adjacency):
        mixed = torch.einsum('ij,bjc->bic', adjacency, node_features)
        return self.act(self.norm(self.linear(mixed) + node_features))

class ACFNetPlusRegionalFeatures(nn.Module):
    def __init__(
        self,
        in_channels=6,
        out_channels=1,
        base_channels=24,
        num_regions=6,
        node_feature_dim=80,
        adjacency=None,
        graph_layers=2,
        dropout=0.15,
    ):
        super().__init__()

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8

        self.num_regions = num_regions

        self.enc0 = ResidualBlock3D(in_channels, c1, dropout=dropout)
        self.down1 = nn.Conv3d(c1, c2, kernel_size=3, stride=2, padding=1)
        self.enc1 = ResidualBlock3D(c2, c2, dropout=dropout)
        self.down2 = nn.Conv3d(c2, c3, kernel_size=3, stride=2, padding=1)
        self.enc2 = ResidualBlock3D(c3, c3, dropout=dropout)
        self.down3 = nn.Conv3d(c3, c4, kernel_size=3, stride=2, padding=1)
        self.enc3 = ResidualBlock3D(c4, c4, dropout=dropout)

        self.node_feature_encoder = nn.Sequential(
            nn.Linear(node_feature_dim, c4),
            nn.LayerNorm(c4),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(c4, c4),
            nn.LayerNorm(c4),
        )

        self.router_feature_encoder = nn.Sequential(
            nn.Linear(node_feature_dim, 24),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout),
        )

        self.graph_layers = nn.ModuleList([GraphConv(c4) for _ in range(graph_layers)])
        self.graph_project = nn.Sequential(
            nn.Linear(c4, c4),
            nn.LayerNorm(c4),
            nn.LeakyReLU(0.1, inplace=True),
        )

        self.up2 = nn.ConvTranspose3d(c4, c3, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock3D(c3 + c3, c3, dropout=dropout)
        self.up1 = nn.ConvTranspose3d(c3, c2, kernel_size=2, stride=2)
        self.dec1 = ResidualBlock3D(c2 + c2, c2, dropout=dropout)
        self.up0 = nn.ConvTranspose3d(c2, c1, kernel_size=2, stride=2)
        self.dec0 = ResidualBlock3D(c1 + c1, c1, dropout=dropout)

        self.context_head = nn.Conv3d(c1, out_channels, kernel_size=1)
        self.boundary_head = nn.Conv3d(c1, out_channels, kernel_size=1)
        self.region_head = nn.Conv3d(c1, out_channels, kernel_size=1)

        self.router = nn.Sequential(
            nn.Linear(num_regions + 4 + 24, 64),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 3),
        )

        if adjacency is None:
            adjacency = torch.eye(num_regions, dtype=torch.float32)

        self.register_buffer('adjacency', adjacency.float())

    def _region_one_hot(self, region, spatial_size):
        region_resized = F.interpolate(
            region[:, None].float(),
            size=spatial_size,
            mode='nearest',
        ).long().squeeze(1)

        region_resized = region_resized.clamp(0, self.num_regions - 1)

        return F.one_hot(
            region_resized,
            num_classes=self.num_regions,
        ).permute(0, 4, 1, 2, 3).float()

    def _pool_regions(self, features, one_hot):
        denom = one_hot.sum(dim=(2, 3, 4)).clamp_min(1.0)
        pooled = torch.einsum('bchwd,brhwd->brc', features, one_hot)
        return pooled / denom[:, :, None]

    def _router_weights(self, region, expert_logits, node_features):
        full_one_hot = self._region_one_hot(region, expert_logits[0].shape[2:])
        region_fraction = full_one_hot.mean(dim=(2, 3, 4))[:, :, None]

        expert_conf = []
        for logits in expert_logits:
            prob = torch.sigmoid(logits.float())
            pooled_prob = self._pool_regions(prob, full_one_hot)
            expert_conf.append(pooled_prob[:, :, 0])

        expert_conf = torch.stack(expert_conf, dim=-1)

        identity = torch.eye(
            self.num_regions,
            device=region.device,
            dtype=torch.float32,
        )[None].repeat(region.shape[0], 1, 1)

        router_features = self.router_feature_encoder(node_features.float())

        router_input = torch.cat(
            [identity, expert_conf, region_fraction, router_features],
            dim=-1,
        )

        router_weights = torch.softmax(self.router(router_input.float()), dim=-1)
        return router_weights, full_one_hot

    def forward(self, image, region, node_features):
        x0 = self.enc0(image.float())
        x1 = self.enc1(self.down1(x0))
        x2 = self.enc2(self.down2(x1))
        x3 = self.enc3(self.down3(x2))

        bottleneck_one_hot = self._region_one_hot(region, x3.shape[2:])
        cnn_node_features = self._pool_regions(x3, bottleneck_one_hot)
        sidecar_node_features = self.node_feature_encoder(node_features.float())

        graph_node_features = cnn_node_features + sidecar_node_features

        for layer in self.graph_layers:
            graph_node_features = layer(graph_node_features, self.adjacency)

        graph_node_features = self.graph_project(graph_node_features)
        graph_volume = torch.einsum('brc,brhwd->bchwd', graph_node_features, bottleneck_one_hot)
        x3 = x3 + graph_volume

        y2 = self.up2(x3)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))

        y1 = self.up1(y2)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))

        y0 = self.up0(y1)
        y0 = self.dec0(torch.cat([y0, x0], dim=1))

        context_logits = self.context_head(y0)
        boundary_logits = self.boundary_head(y0)
        region_logits = self.region_head(y0)

        router_weights, full_one_hot = self._router_weights(
            region,
            [context_logits, boundary_logits, region_logits],
            node_features,
        )

        expert_maps = torch.stack(
            [context_logits[:, 0], boundary_logits[:, 0], region_logits[:, 0]],
            dim=1,
        )

        weight_maps = torch.einsum('bre,brhwd->behwd', router_weights, full_one_hot)
        logits = (expert_maps * weight_maps).sum(dim=1, keepdim=True)

        return {
            'logits': logits,
            'context_logits': context_logits,
            'boundary_logits': boundary_logits,
            'region_logits': region_logits,
            'router_weights': router_weights,
        }

train_dataset = ISLESRegionalFeatureGraphPatchDataset(
    train_nonempty_df,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = ACFNetPlusRegionalFeatures(
    in_channels=6,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    num_regions=NUM_REGIONS,
    node_feature_dim=NODE_FEATURE_DIM,
    adjacency=ADJACENCY,
    graph_layers=2,
    dropout=DROPOUT,
).to(device)

loss_fn = DiceFocalLoss(
    sigmoid=True,
    squared_pred=True,
    lambda_dice=1.0,
    lambda_focal=1.0,
    gamma=2.0,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
)

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'regional_feature_version': REGIONAL_FEATURE_VERSION,
    'model': 'ACF-Net+ coarse 6-region graph with regional sidecar node features',
    'base_channels': BASE_CHANNELS,
    'dropout': DROPOUT,
    'node_feature_dim': NODE_FEATURE_DIM,
    'num_regions': NUM_REGIONS,
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'aux_loss_weight': AUX_LOSS_WEIGHT,
    'loss': 'DiceFocalLoss',
    'max_epochs': MAX_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'validation_threshold': VAL_THRESHOLD,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'seed': SEED,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial5 from: {latest_checkpoint_path}')
    print(f'Start epoch: {start_epoch}')
    print(f'Best val Dice so far: {best_val_dice:.4f}')
else:
    print('Starting new Trial5 regional-feature ACF-Net+ run.')

batch = next(iter(train_loader))
images = batch['image'].to(device)
targets = batch['target'].to(device)
regions = batch['region'].to(device)
node_features = batch['node_features'].to(device)

model.eval()
with torch.no_grad():
    outputs = model(images, regions, node_features)

print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch region:', batch['region'].shape, batch['region'].dtype)
print('Batch node features:', batch['node_features'].shape, batch['node_features'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))
print('Output logits:', outputs['logits'].shape)
print('Router weights:', outputs['router_weights'].shape)
print('Mean router weights:', outputs['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print(f'Trial5 trainable parameters: {count_parameters(model):,}')
print(f'Validation threshold: {VAL_THRESHOLD}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
model.train()

batch = next(iter(train_loader))
images = batch['image'].to(device)
targets = batch['target'].to(device)
regions = batch['region'].to(device)
node_features = batch['node_features'].to(device)

optimizer.zero_grad(set_to_none=True)

outputs = model(images, regions, node_features)

main_loss = loss_fn(outputs['logits'].float(), targets.float())
aux_loss = (
    loss_fn(outputs['context_logits'].float(), targets.float())
    + loss_fn(outputs['boundary_logits'].float(), targets.float())
    + loss_fn(outputs['region_logits'].float(), targets.float())
) / 3.0

loss = main_loss + AUX_LOSS_WEIGHT * aux_loss

print('Preflight loss:', float(loss.detach().cpu()))
print('Preflight main loss:', float(main_loss.detach().cpu()))
print('Preflight aux loss:', float(aux_loss.detach().cpu()))
print('Preflight router:', outputs['router_weights'].detach().mean(dim=(0, 1)).cpu().numpy())

assert torch.isfinite(loss), 'Preflight failed: loss is NaN or Inf'
assert torch.isfinite(outputs['logits']).all(), 'Preflight failed: logits contain NaN or Inf'
assert torch.isfinite(outputs['router_weights']).all(), 'Preflight failed: router contains NaN or Inf'

loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

print('Preflight grad norm:', float(grad_norm.detach().cpu()))
assert torch.isfinite(grad_norm), 'Preflight failed: gradient norm is NaN or Inf'

optimizer.zero_grad(set_to_none=True)

print('Trial5 preflight passed.')


In [ ]:
import time
GRAD_CLIP_NORM = 1.0

def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def load_full_case_npz(npz_path):
    arr = np.load(npz_path, allow_pickle=True)
    image = arr['image'].astype(np.float32)
    target = arr['target'].astype(np.uint8)
    return image, target

def load_normed_node_features(path):
    node_features = load_regional_features(path)
    node_features = (node_features - feature_mean[0]) / feature_std[0]
    node_features = np.nan_to_num(
        node_features,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(np.float32)
    return node_features

def pad_image_target_region_to_roi(image, target, region, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if pad_h or pad_w or pad_d:
        spatial_pad = (
            (pad_h // 2, pad_h - pad_h // 2),
            (pad_w // 2, pad_w - pad_w // 2),
            (pad_d // 2, pad_d - pad_d // 2),
        )

        image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant')
        target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant')
        region = np.pad(region, spatial_pad, mode='constant')

    return image, target, region

@torch.no_grad()
def predict_full_volume_trial5_prob(model, image, region, node_features, roi_size):
    model.eval()

    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    logits_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)
    count_map = torch.zeros((1, 1, h, w, d), dtype=torch.float32, device=device)

    router_rows = []

    h_starts = list(range(0, max(h - ph, 0) + 1, ph // 2))
    w_starts = list(range(0, max(w - pw, 0) + 1, pw // 2))
    d_starts = list(range(0, max(d - pd_, 0) + 1, pd_ // 2))

    if h_starts[-1] != h - ph:
        h_starts.append(h - ph)
    if w_starts[-1] != w - pw:
        w_starts.append(w - pw)
    if d_starts[-1] != d - pd_:
        d_starts.append(d - pd_)

    image_t = torch.from_numpy(image).to(device)
    region_t = torch.from_numpy(region).long().to(device)
    node_features_t = torch.from_numpy(node_features).to(device)[None]

    for sh in h_starts:
        for sw in w_starts:
            for sd in d_starts:
                image_patch = image_t[None, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_]
                region_patch = region_t[None, sh:sh+ph, sw:sw+pw, sd:sd+pd_]

                outputs = model(image_patch, region_patch, node_features_t)
                logits_patch = outputs['logits']

                if not torch.isfinite(logits_patch).all():
                    raise RuntimeError('Validation logits contain NaN or Inf.')

                logits_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += logits_patch
                count_map[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += 1

                router_rows.append(
                    outputs['router_weights']
                    .detach()
                    .mean(dim=(0, 1))
                    .cpu()
                    .numpy()
                )

    logits = logits_sum / torch.clamp(count_map, min=1.0)
    prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
    mean_router = np.vstack(router_rows).mean(axis=0)

    return prob, mean_router

@torch.no_grad()
def validate_full_volume_trial5(model, val_df):
    model.eval()

    dice_rows = []

    for case_idx, (_, row) in enumerate(val_df.iterrows(), start=1):
        sid = row['subject_id']

        image, target = load_full_case_npz(row['preprocessed_npz_path'])
        region = build_region_map(image.shape[1:])
        node_features = load_normed_node_features(row['regional_feature_path'])

        image, target, region = pad_image_target_region_to_roi(
            image=image,
            target=target,
            region=region,
            roi_size=PATCH_SIZE,
        )

        prob, mean_router = predict_full_volume_trial5_prob(
            model=model,
            image=image,
            region=region,
            node_features=node_features,
            roi_size=PATCH_SIZE,
        )

        pred = prob >= VAL_THRESHOLD
        target_np = target[0] > 0

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())
        dice = dice_score_binary(pred, target_np)

        dice_rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
            'lesion_volume_bin': row['lesion_volume_bin'],
            'router_context': float(mean_router[0]),
            'router_boundary': float(mean_router[1]),
            'router_region': float(mean_router[2]),
        })

        print(
            f'  Val {case_idx}/{len(val_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}, '
            f'router={mean_router}'
        )

    dice_df = pd.DataFrame(dice_rows)
    nonempty_df = dice_df[~dice_df['target_empty']]

    summary = {
        'mean_dice_all': float(dice_df['dice'].mean()),
        'mean_dice_nonempty': float(nonempty_df['dice'].mean()),
        'num_empty_targets': int(dice_df['target_empty'].sum()),
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()),
        'mean_router_context': float(dice_df['router_context'].mean()),
        'mean_router_boundary': float(dice_df['router_boundary'].mean()),
        'mean_router_region': float(dice_df['router_region'].mean()),
    }

    return summary, dice_df

def compute_trial5_loss(outputs, targets):
    targets = targets.float()

    main_loss = loss_fn(outputs['logits'].float(), targets)

    aux_loss = (
        loss_fn(outputs['context_logits'].float(), targets)
        + loss_fn(outputs['boundary_logits'].float(), targets)
        + loss_fn(outputs['region_logits'].float(), targets)
    ) / 3.0

    total_loss = main_loss + AUX_LOSS_WEIGHT * aux_loss

    return total_loss, {
        'main_loss': float(main_loss.detach().cpu()),
        'aux_loss': float(aux_loss.detach().cpu()),
    }

def save_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
        'feature_mean': feature_mean,
        'feature_std': feature_std,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch_trial5(epoch):
    model.train()

    losses = []
    main_losses = []
    aux_losses = []
    router_rows = []

    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        regions = batch['region'].to(device, non_blocking=True)
        node_features = batch['node_features'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images, regions, node_features)
        loss, parts = compute_trial5_loss(outputs, targets)

        if not torch.isfinite(loss):
            print('Invalid loss detected.')
            print('Loss parts:', parts)
            print('Batch subject:', batch['subject_id'])
            print('Router:', outputs['router_weights'].detach().mean(dim=(0, 1)).cpu().numpy())
            raise RuntimeError('Stopping Trial5 because loss became NaN or Inf.')

        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        if not torch.isfinite(grad_norm):
            raise RuntimeError('Stopping Trial5 because gradient norm became NaN or Inf.')

        optimizer.step()

        losses.append(float(loss.detach().cpu()))
        main_losses.append(parts['main_loss'])
        aux_losses.append(parts['aux_loss'])

        mean_router = outputs['router_weights'].detach().mean(dim=(0, 1)).cpu().numpy()
        router_rows.append(mean_router)

        if step == 1 or step % 50 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | Step {step:04d}/{len(train_loader)} | '
                f'Loss {losses[-1]:.4f} | Main {main_losses[-1]:.4f} | '
                f'Aux {aux_losses[-1]:.4f} | Grad {float(grad_norm):.4f} | '
                f'Router {mean_router}'
            )

    elapsed = time.time() - start_time
    router_mean = np.vstack(router_rows).mean(axis=0)

    return {
        'train_loss': float(np.mean(losses)),
        'train_main_loss': float(np.mean(main_losses)),
        'train_aux_loss': float(np.mean(aux_losses)),
        'train_router_context': float(router_mean[0]),
        'train_router_boundary': float(router_mean[1]),
        'train_router_region': float(router_mean[2]),
        'elapsed_seconds': elapsed,
    }

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'{TRAINING_RUN_NAME} Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_summary = train_one_epoch_trial5(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning Trial5 validation...')
        val_summary, val_df_epoch = validate_full_volume_trial5(model, val_df)

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(
            'Validation mean router weights: '
            f"context={val_summary['mean_router_context']:.4f}, "
            f"boundary={val_summary['mean_router_boundary']:.4f}, "
            f"region={val_summary['mean_router_region']:.4f}"
        )
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    history_row = {
        'epoch': epoch,
        **train_summary,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'val_router_context': None if val_summary is None else val_summary['mean_router_context'],
        'val_router_boundary': None if val_summary is None else val_summary['mean_router_boundary'],
        'val_router_region': None if val_summary is None else val_summary['mean_router_region'],
        'lr': current_lr,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Trial5 epoch {epoch} complete.')
    print(f"Train loss: {train_summary['train_loss']:.4f}")
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'LR: {current_lr:.8f}')
    print(f"Elapsed: {train_summary['elapsed_seconds'] / 60:.2f} min")
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after '
            f'{epochs_without_improvement} epochs without validation improvement.'
        )
        break

print('Trial5 training complete.')


In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from google.colab import drive
from monai.networks.nets import SegResNet
from monai.losses import DiceFocalLoss

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()
    if project_root not in path.parents and path != project_root:
        raise ValueError(f'Unsafe path outside project root: {path}')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'
TRAINING_RUN_NAME = 'trial6_segresnet_gnn_expert_adapter_v1'

SEED = 20240824
PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1536
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.90

MAX_EPOCHS = 120
MIN_EPOCHS_BEFORE_STOPPING = 40
EARLY_STOPPING_PATIENCE = 25
MIN_DELTA = 1e-4

MAIN_LR = 5e-5
ADAPTER_LR = 2e-4
WEIGHT_DECAY = 1e-5
VAL_EVERY = 1
VAL_THRESHOLD = 0.20
USE_AMP = False
GRAD_CLIP_NORM = 1.0

NUM_REGIONS = 6
NODE_FEATURE_DIM = 80
GRAPH_DIM = 64
NUM_EXPERTS = 3
ADAPTER_CHANNELS = 32
ADAPTER_INIT_SCALE = 0.05
AUX_MAIN_LOSS_WEIGHT = 0.30
RESIDUAL_REG_WEIGHT = 1e-4

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')

MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
REGIONAL_FEATURE_DIR = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION / 'coarse6_regional_features_v1'
REGIONAL_FEATURE_AUDIT_PATH = REGIONAL_FEATURE_DIR / 'regional_feature_audit.csv'

TRIAL2_BEST_CHECKPOINT_PATH = (
    GENERATED_DIRS['checkpoints']
    / 'trial2_segresnet_imbalance_tuned_v1'
    / 'best.pt'
)

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)
regional_audit_df = pd.read_csv(REGIONAL_FEATURE_AUDIT_PATH)

valid_regional_df = regional_audit_df[
    (regional_audit_df['readable'])
    & (regional_audit_df['finite'])
    & (regional_audit_df['num_regions'] == NUM_REGIONS)
    & (regional_audit_df['num_features'] == NODE_FEATURE_DIM)
].copy()

regional_feature_path_by_subject = dict(
    zip(valid_regional_df['subject_id'], valid_regional_df['regional_feature_path'])
)

model_manifest_df = model_manifest_df[
    model_manifest_df['subject_id'].isin(regional_feature_path_by_subject)
].reset_index(drop=True)

train_df = model_manifest_df[model_manifest_df['dev_split'] == 'train'].reset_index(drop=True)
val_df = model_manifest_df[model_manifest_df['dev_split'] == 'val'].reset_index(drop=True)
test_df = model_manifest_df[model_manifest_df['dev_split'] == 'test'].reset_index(drop=True)

train_nonempty_df = train_df[
    train_df['preprocessed_target_nonzero_voxels'] > 0
].reset_index(drop=True)

print(f'Total manifest subjects with sidecars: {len(model_manifest_df)}')
print(f'Original train subjects: {len(train_df)}')
print(f'Non-empty train subjects for Trial6: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')
print(
    'Excluded empty train subjects:',
    sorted(set(train_df['subject_id']) - set(train_nonempty_df['subject_id']))
)

REGION_NAMES = [
    'left_anterior',
    'left_middle',
    'left_posterior',
    'right_anterior',
    'right_middle',
    'right_posterior',
]

def make_adjacency():
    edges = [
        (0, 1), (1, 2),
        (3, 4), (4, 5),
        (0, 3), (1, 4), (2, 5),
    ]

    adjacency = torch.eye(NUM_REGIONS, dtype=torch.float32)

    for a, b in edges:
        adjacency[a, b] = 1.0
        adjacency[b, a] = 1.0

    adjacency = adjacency / adjacency.sum(dim=1, keepdim=True)
    return adjacency

ADJACENCY = make_adjacency().to(device)

print('Region names:', REGION_NAMES)
print('Adjacency:', ADJACENCY.detach().cpu())

def make_coarse6_region_map(spatial_shape):
    h, w, d = spatial_shape
    region = np.zeros((h, w, d), dtype=np.int64)

    midline = h // 2
    anterior_end = w // 3
    middle_end = (2 * w) // 3

    region[:midline, :anterior_end, :] = 0
    region[:midline, anterior_end:middle_end, :] = 1
    region[:midline, middle_end:, :] = 2

    region[midline:, :anterior_end, :] = 3
    region[midline:, anterior_end:middle_end, :] = 4
    region[midline:, middle_end:, :] = 5

    return region

example_arr = np.load(train_nonempty_df.iloc[0]['preprocessed_npz_path'], allow_pickle=True)
example_image = example_arr['image']
example_region = make_coarse6_region_map(example_image.shape[1:])

print('Example image shape:', example_image.shape)
print('Example region shape:', example_region.shape)
print('Unique regions:', np.unique(example_region))
print('Region voxel counts:', np.bincount(example_region.reshape(-1), minlength=NUM_REGIONS))

assert example_region.shape == example_image.shape[1:]
assert set(np.unique(example_region).tolist()) == set(range(NUM_REGIONS))

print('Region map smoke check passed.')

def load_regional_features(path):
    arr = np.load(path, allow_pickle=True)

    preferred_keys = [
        'features',
        'regional_features',
        'node_features',
        'region_features',
    ]

    for key in preferred_keys:
        if key in arr.files:
            value = arr[key]
            if value.ndim == 2 and value.shape[0] == NUM_REGIONS:
                return value.astype(np.float32)

    for key in arr.files:
        value = arr[key]
        if value.ndim == 2 and value.shape[0] == NUM_REGIONS:
            return value.astype(np.float32)

    raise ValueError(f'No usable regional feature matrix found in {path}')

class ISLESTrial6PatchDataset(Dataset):
    def __init__(
        self,
        df,
        regional_feature_path_by_subject,
        patch_size=(128, 128, 64),
        samples_per_epoch=1536,
        positive_prob=0.90,
        seed=20240824,
        balance_by_lesion_bin=True,
    ):
        self.df = df.reset_index(drop=True)
        self.regional_feature_path_by_subject = regional_feature_path_by_subject
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch
        self.positive_prob = positive_prob
        self.balance_by_lesion_bin = balance_by_lesion_bin
        self.rng = np.random.default_rng(seed)

        self.bin_to_indices = {
            lesion_bin: group.index.to_numpy()
            for lesion_bin, group in self.df.groupby('lesion_volume_bin')
        }

        requested_weights = {
            'tiny': 0.22,
            'small': 0.26,
            'medium': 0.26,
            'large': 0.16,
            'very_large': 0.10,
        }

        self.available_bins = [
            b for b in requested_weights
            if b in self.bin_to_indices and len(self.bin_to_indices[b]) > 0
        ]

        raw_weights = np.array(
            [requested_weights[b] for b in self.available_bins],
            dtype=np.float64,
        )
        self.bin_probs = raw_weights / raw_weights.sum()

    def __len__(self):
        return self.samples_per_epoch

    def _choose_row(self, index):
        if not self.balance_by_lesion_bin:
            return self.df.iloc[index % len(self.df)]

        lesion_bin = self.rng.choice(self.available_bins, p=self.bin_probs)
        case_index = int(self.rng.choice(self.bin_to_indices[lesion_bin]))
        return self.df.iloc[case_index]

    def _pad_to_patch(self, image, target, region):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            spatial_pad = (
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )

            image = np.pad(
                image,
                ((0, 0),) + spatial_pad,
                mode='constant',
                constant_values=0,
            )
            target = np.pad(
                target,
                ((0, 0),) + spatial_pad,
                mode='constant',
                constant_values=0,
            )
            region = np.pad(region, spatial_pad, mode='edge')

        return image, target, region

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            center = coords[self.rng.integers(0, len(coords))]
            ch, cw, cd = center
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        row = self._choose_row(index)
        subject_id = row['subject_id']

        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)

        region = make_coarse6_region_map(image.shape[1:])
        image, target, region = self._pad_to_patch(image, target, region)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        node_features = load_regional_features(
            self.regional_feature_path_by_subject[subject_id]
        )

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'region': torch.from_numpy(region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'node_features': torch.from_numpy(node_features),
            'subject_id': subject_id,
        }

class SimpleGraphBlock(nn.Module):
    def __init__(self, dim, dropout=0.10):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, adjacency):
        residual = x
        x = torch.einsum('ij,bjf->bif', adjacency, x)
        x = self.norm(x)
        x = F.gelu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return residual + x

class ConvExpert(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(hidden_channels),
            nn.GELU(),
            nn.Conv3d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(hidden_channels),
            nn.GELU(),
            nn.Conv3d(hidden_channels, 1, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x)

class Trial6SegResNetGNNExpertAdapter(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        node_feature_dim,
        num_regions,
        adjacency,
        graph_dim=64,
        num_experts=3,
        adapter_channels=32,
        adapter_init_scale=0.05,
    ):
        super().__init__()

        self.num_regions = num_regions
        self.num_experts = num_experts
        self.register_buffer('adjacency', adjacency.detach().clone())

        self.main_model = SegResNet(
            spatial_dims=3,
            init_filters=16,
            in_channels=in_channels,
            out_channels=out_channels,
            dropout_prob=0.15,
            blocks_down=(1, 2, 2, 4),
            blocks_up=(1, 1, 1),
            norm='INSTANCE',
        )

        self.node_encoder = nn.Sequential(
            nn.LayerNorm(node_feature_dim),
            nn.Linear(node_feature_dim, graph_dim),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(graph_dim, graph_dim),
        )

        self.graph_blocks = nn.ModuleList([
            SimpleGraphBlock(graph_dim, dropout=0.10),
            SimpleGraphBlock(graph_dim, dropout=0.10),
        ])

        self.router = nn.Sequential(
            nn.LayerNorm(graph_dim),
            nn.Linear(graph_dim, graph_dim),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(graph_dim, num_experts),
        )

        self.experts = nn.ModuleList([
            ConvExpert(in_channels, adapter_channels)
            for _ in range(num_experts)
        ])

        init_scale = float(np.clip(adapter_init_scale, 1e-4, 0.99))
        init_logit = np.log(init_scale / (1.0 - init_scale))
        self.adapter_scale_logit = nn.Parameter(
            torch.tensor(init_logit, dtype=torch.float32)
        )

    def _make_router_maps(self, router_weights, region):
        b, h, w, d = region.shape
        maps = []

        for expert_idx in range(self.num_experts):
            weight_map = torch.zeros(
                (b, 1, h, w, d),
                device=region.device,
                dtype=router_weights.dtype,
            )

            for region_idx in range(self.num_regions):
                mask = (region == region_idx).unsqueeze(1)
                value = router_weights[:, region_idx, expert_idx].view(b, 1, 1, 1, 1)
                weight_map = weight_map + mask.to(router_weights.dtype) * value

            maps.append(weight_map)

        return maps

    def forward(self, image, region, node_features):
        main_logits = self.main_model(image)

        node = self.node_encoder(node_features)

        for block in self.graph_blocks:
            node = block(node, self.adjacency)

        router_weights = torch.softmax(self.router(node), dim=-1)
        router_maps = self._make_router_maps(router_weights, region)

        expert_logits = [expert(image) for expert in self.experts]

        residual_logits = torch.zeros_like(main_logits)

        for expert_idx, expert_logit in enumerate(expert_logits):
            residual_logits = residual_logits + expert_logit * router_maps[expert_idx]

        adapter_scale = torch.sigmoid(self.adapter_scale_logit)
        logits = main_logits + adapter_scale * residual_logits

        return {
            'logits': logits,
            'main_logits': main_logits,
            'residual_logits': residual_logits,
            'router_weights': router_weights,
            'adapter_scale': adapter_scale,
        }

train_dataset = ISLESTrial6PatchDataset(
    train_nonempty_df,
    regional_feature_path_by_subject=regional_feature_path_by_subject,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
    balance_by_lesion_bin=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = Trial6SegResNetGNNExpertAdapter(
    in_channels=6,
    out_channels=1,
    node_feature_dim=NODE_FEATURE_DIM,
    num_regions=NUM_REGIONS,
    adjacency=ADJACENCY,
    graph_dim=GRAPH_DIM,
    num_experts=NUM_EXPERTS,
    adapter_channels=ADAPTER_CHANNELS,
    adapter_init_scale=ADAPTER_INIT_SCALE,
).to(device)

loss_fn = DiceFocalLoss(
    sigmoid=True,
    squared_pred=True,
    lambda_dice=1.0,
    lambda_focal=1.0,
    gamma=2.0,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)

main_params = list(model.main_model.parameters())
adapter_params = [
    p for name, p in model.named_parameters()
    if not name.startswith('main_model.')
]

optimizer = torch.optim.AdamW(
    [
        {'params': main_params, 'lr': MAIN_LR},
        {'params': adapter_params, 'lr': ADAPTER_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=20,
    T_mult=2,
    eta_min=1e-6,
)

try:
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and torch.cuda.is_available())
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available())

def autocast_context():
    if USE_AMP and torch.cuda.is_available():
        return torch.amp.autocast('cuda')
    return nullcontext()

def clean_state_dict_keys(state_dict):
    cleaned = {}

    for key, value in state_dict.items():
        new_key = key
        if new_key.startswith('module.'):
            new_key = new_key[len('module.'):]
        if new_key.startswith('main_model.'):
            new_key = new_key[len('main_model.'):]
        cleaned[new_key] = value

    return cleaned

def load_trial2_main_path(model, checkpoint_path):
    if not checkpoint_path.exists():
        print(f'Trial2 checkpoint not found, starting main path from scratch: {checkpoint_path}')
        return

    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state = checkpoint.get('model_state_dict', checkpoint)
    state = clean_state_dict_keys(state)

    try:
        model.main_model.load_state_dict(state, strict=True)
        print(f'Loaded Trial2 main path strictly from: {checkpoint_path}')
        return
    except RuntimeError as err:
        print('Strict Trial2 load failed. Trying shape-matched partial load.')
        print(str(err).split('\n')[0])

    current = model.main_model.state_dict()
    matched = {
        key: value
        for key, value in state.items()
        if key in current and current[key].shape == value.shape
    }

    current.update(matched)
    model.main_model.load_state_dict(current, strict=True)

    print(f'Loaded {len(matched)}/{len(current)} Trial2-compatible tensors into Trial6 main path.')

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'trial_note': 'Trial2 SegResNet main path plus separated GNN-conditioned expert residual adapter.',
    'trial2_checkpoint_source': str(TRIAL2_BEST_CHECKPOINT_PATH),
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'train_subjects': len(train_nonempty_df),
    'excluded_empty_train_subjects': sorted(
        set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])
    ),
    'max_epochs': MAX_EPOCHS,
    'min_epochs_before_stopping': MIN_EPOCHS_BEFORE_STOPPING,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'main_lr': MAIN_LR,
    'adapter_lr': ADAPTER_LR,
    'weight_decay': WEIGHT_DECAY,
    'validation_threshold': VAL_THRESHOLD,
    'loss': 'DiceFocalLoss + auxiliary main-path loss + residual regularization',
    'scheduler': 'CosineAnnealingWarmRestarts',
    'node_feature_dim': NODE_FEATURE_DIM,
    'graph_dim': GRAPH_DIM,
    'num_experts': NUM_EXPERTS,
    'adapter_channels': ADAPTER_CHANNELS,
    'adapter_init_scale': ADAPTER_INIT_SCALE,
    'seed': SEED,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial6 from: {latest_checkpoint_path}')
    print(f'Start epoch: {start_epoch}')
    print(f'Best val Dice so far: {best_val_dice:.4f}')
    print(f'Epochs without improvement: {epochs_without_improvement}')
else:
    print('Starting new Trial6 run.')
    load_trial2_main_path(model, TRIAL2_BEST_CHECKPOINT_PATH)

batch = next(iter(train_loader))
model.eval()

with torch.no_grad():
    output = model(
        batch['image'].to(device),
        batch['region'].to(device),
        batch['node_features'].to(device),
    )

total_params, trainable_params = count_parameters(model)

print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch region:', batch['region'].shape, batch['region'].dtype)
print('Batch node features:', batch['node_features'].shape, batch['node_features'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))
print('Output logits:', output['logits'].shape)
print('Main logits:', output['main_logits'].shape)
print('Residual logits:', output['residual_logits'].shape)
print('Router weights:', output['router_weights'].shape)
print('Mean router weights:', output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print('Adapter scale:', float(output['adapter_scale'].detach().cpu()))
print(f'Trial6 total parameters: {total_params:,}')
print(f'Trial6 trainable parameters: {trainable_params:,}')
print(f'Validation threshold: {VAL_THRESHOLD}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
model.train()
batch = next(iter(train_loader))

images = batch['image'].to(device, non_blocking=True)
targets = batch['target'].to(device, non_blocking=True)
regions = batch['region'].to(device, non_blocking=True)
node_features = batch['node_features'].to(device, non_blocking=True)

optimizer.zero_grad(set_to_none=True)

with autocast_context():
    output = model(images, regions, node_features)

    main_loss = loss_fn(output['main_logits'], targets)
    final_loss = loss_fn(output['logits'], targets)
    residual_reg = output['residual_logits'].pow(2).mean()

    loss = (
        final_loss
        + AUX_MAIN_LOSS_WEIGHT * main_loss
        + RESIDUAL_REG_WEIGHT * residual_reg
    )

if not torch.isfinite(loss):
    raise RuntimeError(f'Non-finite Trial6 preflight loss: {loss.item()}')

scaler.scale(loss).backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=GRAD_CLIP_NORM,
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

print('Preflight loss:', float(loss.detach().cpu()))
print('Preflight final loss:', float(final_loss.detach().cpu()))
print('Preflight main loss:', float(main_loss.detach().cpu()))
print('Preflight residual reg:', float(residual_reg.detach().cpu()))
print('Preflight grad norm:', float(grad_norm.detach().cpu()))
print('Preflight router:', output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print('Preflight adapter scale:', float(output['adapter_scale'].detach().cpu()))
print('Trial6 preflight passed.')


In [ ]:
def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def pad_case_to_roi(image, target, region, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if not (pad_h or pad_w or pad_d):
        return image, target, region, (h, w, d)

    spatial_pad = (
        (pad_h // 2, pad_h - pad_h // 2),
        (pad_w // 2, pad_w - pad_w // 2),
        (pad_d // 2, pad_d - pad_d // 2),
    )

    image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant', constant_values=0)
    target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant', constant_values=0)
    region = np.pad(region, spatial_pad, mode='edge')

    return image, target, region, (h, w, d)

def get_scan_starts(length, roi, overlap):
    if length <= roi:
        return [0]

    stride = max(1, int(roi * (1.0 - overlap)))
    starts = list(range(0, length - roi + 1, stride))

    if starts[-1] != length - roi:
        starts.append(length - roi)

    return starts

@torch.no_grad()
def trial6_sliding_window_predict(model, image, region, node_features, roi_size, overlap=0.5):
    model.eval()

    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    h_starts = get_scan_starts(h, ph, overlap)
    w_starts = get_scan_starts(w, pw, overlap)
    d_starts = get_scan_starts(d, pd_, overlap)

    logit_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32)
    count_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32)

    router_values = []
    adapter_scales = []

    node_t = torch.from_numpy(node_features[None]).to(device)

    for sh in h_starts:
        for sw in w_starts:
            for sd in d_starts:
                image_patch = image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]
                region_patch = region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]

                image_t = torch.from_numpy(image_patch[None]).to(device)
                region_t = torch.from_numpy(region_patch[None]).to(device)

                output = model(image_t, region_t, node_t)
                logits = output['logits'].detach().cpu()

                logit_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += logits
                count_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += 1.0

                router_values.append(
                    output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy()
                )
                adapter_scales.append(float(output['adapter_scale'].detach().cpu()))

    logits = logit_sum / torch.clamp(count_sum, min=1.0)
    prob = torch.sigmoid(logits)[0, 0].numpy()

    router_mean = np.mean(np.stack(router_values, axis=0), axis=0)
    adapter_scale_mean = float(np.mean(adapter_scales))

    return prob, router_mean, adapter_scale_mean

@torch.no_grad()
def validate_trial6(model, val_df, threshold=VAL_THRESHOLD, max_subjects=None):
    model.eval()

    rows = []
    eval_df = val_df if max_subjects is None else val_df.head(max_subjects)

    for eval_idx, (_, row) in enumerate(eval_df.iterrows(), start=1):
        sid = row['subject_id']

        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.uint8)
        region = make_coarse6_region_map(image.shape[1:])
        node_features = load_regional_features(regional_feature_path_by_subject[sid])

        image, target, region, original_shape = pad_case_to_roi(
            image,
            target,
            region,
            PATCH_SIZE,
        )

        prob, router_mean, adapter_scale = trial6_sliding_window_predict(
            model,
            image,
            region,
            node_features,
            roi_size=PATCH_SIZE,
            overlap=0.5,
        )

        oh, ow, od = original_shape
        prob = prob[:oh, :ow, :od]
        target_np = target[0, :oh, :ow, :od] > 0

        pred = prob >= threshold
        dice = dice_score_binary(pred, target_np)

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())

        rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
            'lesion_volume_bin': row['lesion_volume_bin'],
            'router_expert0': float(router_mean[0]),
            'router_expert1': float(router_mean[1]),
            'router_expert2': float(router_mean[2]),
            'adapter_scale': adapter_scale,
        })

        print(
            f'  Val {eval_idx}/{len(eval_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}, '
            f'scale={adapter_scale:.4f}'
        )

    dice_df = pd.DataFrame(rows)
    nonempty_df = dice_df[~dice_df['target_empty']]

    summary = {
        'mean_dice_all': float(dice_df['dice'].mean()) if len(dice_df) else 0.0,
        'mean_dice_nonempty': float(nonempty_df['dice'].mean()) if len(nonempty_df) else 0.0,
        'num_cases': int(len(dice_df)),
        'num_empty_targets': int(dice_df['target_empty'].sum()) if len(dice_df) else 0,
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()) if len(dice_df) else 0.0,
        'mean_router_expert0': float(dice_df['router_expert0'].mean()) if len(dice_df) else 0.0,
        'mean_router_expert1': float(dice_df['router_expert1'].mean()) if len(dice_df) else 0.0,
        'mean_router_expert2': float(dice_df['router_expert2'].mean()) if len(dice_df) else 0.0,
        'mean_adapter_scale': float(dice_df['adapter_scale'].mean()) if len(dice_df) else 0.0,
    }

    return summary, dice_df

def save_trial6_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch_trial6(epoch):
    model.train()

    losses = []
    final_losses = []
    main_losses = []
    residual_regs = []
    router_rows = []
    adapter_scales = []

    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        regions = batch['region'].to(device, non_blocking=True)
        node_features = batch['node_features'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast_context():
            output = model(images, regions, node_features)

            main_loss = loss_fn(output['main_logits'], targets)
            final_loss = loss_fn(output['logits'], targets)
            residual_reg = output['residual_logits'].pow(2).mean()

            loss = (
                final_loss
                + AUX_MAIN_LOSS_WEIGHT * main_loss
                + RESIDUAL_REG_WEIGHT * residual_reg
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f'Non-finite loss at epoch {epoch}, step {step}: {float(loss.detach().cpu())}'
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM,
        )

        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.detach().cpu()))
        final_losses.append(float(final_loss.detach().cpu()))
        main_losses.append(float(main_loss.detach().cpu()))
        residual_regs.append(float(residual_reg.detach().cpu()))
        router_rows.append(output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
        adapter_scales.append(float(output['adapter_scale'].detach().cpu()))

        if step == 1 or step % 75 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | Step {step:04d}/{len(train_loader)} | '
                f'Loss {losses[-1]:.4f} | Final {final_losses[-1]:.4f} | '
                f'Main {main_losses[-1]:.4f} | Scale {adapter_scales[-1]:.4f} | '
                f'Grad {float(grad_norm.detach().cpu()):.4f}'
            )

    router_mean = np.mean(np.stack(router_rows, axis=0), axis=0)
    elapsed = time.time() - start_time

    return {
        'train_loss': float(np.mean(losses)),
        'train_final_loss': float(np.mean(final_losses)),
        'train_main_loss': float(np.mean(main_losses)),
        'train_residual_reg': float(np.mean(residual_regs)),
        'train_router_expert0': float(router_mean[0]),
        'train_router_expert1': float(router_mean[1]),
        'train_router_expert2': float(router_mean[2]),
        'train_adapter_scale': float(np.mean(adapter_scales)),
        'elapsed_seconds': elapsed,
    }

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'{TRAINING_RUN_NAME} Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_summary = train_one_epoch_trial6(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning Trial6 validation...')
        val_summary, val_df_epoch = validate_trial6(
            model,
            val_df,
            threshold=VAL_THRESHOLD,
        )

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(
            'Validation router means: '
            f"{val_summary['mean_router_expert0']:.4f}, "
            f"{val_summary['mean_router_expert1']:.4f}, "
            f"{val_summary['mean_router_expert2']:.4f}"
        )
        print(f"Validation adapter scale:         {val_summary['mean_adapter_scale']:.4f}")
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step(epoch + 1)

    current_main_lr = optimizer.param_groups[0]['lr']
    current_adapter_lr = optimizer.param_groups[1]['lr']

    history_row = {
        'epoch': epoch,
        **train_summary,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'val_router_expert0': None if val_summary is None else val_summary['mean_router_expert0'],
        'val_router_expert1': None if val_summary is None else val_summary['mean_router_expert1'],
        'val_router_expert2': None if val_summary is None else val_summary['mean_router_expert2'],
        'val_adapter_scale': None if val_summary is None else val_summary['mean_adapter_scale'],
        'main_lr': current_main_lr,
        'adapter_lr': current_adapter_lr,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best Trial6 validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            epochs_without_improvement += 1
            print(
                f'No validation improvement for '
                f'{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epochs.'
            )

    save_trial6_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Epoch {epoch} complete.')
    print(f"Train loss: {train_summary['train_loss']:.4f}")
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'Main LR: {current_main_lr:.8f}')
    print(f'Adapter LR: {current_adapter_lr:.8f}')
    print(f"Elapsed: {train_summary['elapsed_seconds'] / 60:.2f} min")
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    can_stop = epoch + 1 >= MIN_EPOCHS_BEFORE_STOPPING

    if can_stop and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after epoch {epoch}: '
            f'{epochs_without_improvement} epochs without validation improvement, '
            f'with minimum epoch requirement satisfied.'
        )
        break

    if not can_stop:
        print(
            f'Early stopping disabled until epoch {MIN_EPOCHS_BEFORE_STOPPING - 1}.'
        )

print('Trial6 training complete.')


In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import monai
except ImportError:
    !pip -q install monai
    import monai

from google.colab import drive
from monai.networks.nets import SegResNet
from monai.losses import DiceFocalLoss

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()
    if project_root not in path.parents and path != project_root:
        raise ValueError(f'Unsafe path outside project root: {path}')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

PREPROCESSING_VERSION = 'v1_trial0'
SPLIT_NAME = 'trial0_seed20240824_train70_val15_test15'
TRAINING_RUN_NAME = 'trial7_cnn_informed_gnn_router_v1'

SEED = 20240824
PATCH_SIZE = (128, 128, 64)
SAMPLES_PER_TRAIN_EPOCH = 1536
BATCH_SIZE = 1
NUM_WORKERS = 0
POSITIVE_PROB = 0.90

MAX_EPOCHS = 120
FREEZE_MAIN_EPOCHS = 8
MIN_EPOCHS_BEFORE_STOPPING = 45
EARLY_STOPPING_PATIENCE = 25
MIN_DELTA = 1e-4

MAIN_LR = 2e-5
ADAPTER_LR = 2e-4
WEIGHT_DECAY = 1e-5
VAL_EVERY = 1
VAL_THRESHOLD = 0.20
USE_AMP = False
GRAD_CLIP_NORM = 1.0

NUM_REGIONS = 6
NODE_FEATURE_DIM = 80
CNN_EVIDENCE_DIM = 18
GRAPH_DIM = 64
ROUTER_HIDDEN_DIM = 96
NUM_EXPERTS = 3
ADAPTER_CHANNELS = 32

ADAPTER_INIT_SCALE = 0.01
ADAPTER_MAX_SCALE = 0.05
AUX_MAIN_LOSS_WEIGHT = 0.20
RESIDUAL_REG_WEIGHT = 5e-4
ROUTER_ENTROPY_WEIGHT = 1e-4

seed_everything(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}')

MODEL_MANIFEST_PATH = GENERATED_DIRS['metadata'] / 'model_manifest_v1_trial0.csv'
REGIONAL_FEATURE_DIR = GENERATED_DIRS['preprocessing'] / PREPROCESSING_VERSION / 'coarse6_regional_features_v1'
REGIONAL_FEATURE_AUDIT_PATH = REGIONAL_FEATURE_DIR / 'regional_feature_audit.csv'

TRIAL2_BEST_CHECKPOINT_PATH = (
    GENERATED_DIRS['checkpoints']
    / 'trial2_segresnet_imbalance_tuned_v1'
    / 'best.pt'
)

checkpoint_dir = GENERATED_DIRS['checkpoints'] / TRAINING_RUN_NAME
log_dir = GENERATED_DIRS['logs'] / TRAINING_RUN_NAME
output_dir = GENERATED_DIRS['outputs'] / TRAINING_RUN_NAME

for path in [checkpoint_dir, log_dir, output_dir]:
    assert_generated_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

model_manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)
regional_audit_df = pd.read_csv(REGIONAL_FEATURE_AUDIT_PATH)

valid_regional_df = regional_audit_df[
    (regional_audit_df['readable'])
    & (regional_audit_df['finite'])
    & (regional_audit_df['num_regions'] == NUM_REGIONS)
    & (regional_audit_df['num_features'] == NODE_FEATURE_DIM)
].copy()

regional_feature_path_by_subject = dict(
    zip(valid_regional_df['subject_id'], valid_regional_df['regional_feature_path'])
)

model_manifest_df = model_manifest_df[
    model_manifest_df['subject_id'].isin(regional_feature_path_by_subject)
].reset_index(drop=True)

train_df = model_manifest_df[model_manifest_df['dev_split'] == 'train'].reset_index(drop=True)
val_df = model_manifest_df[model_manifest_df['dev_split'] == 'val'].reset_index(drop=True)
test_df = model_manifest_df[model_manifest_df['dev_split'] == 'test'].reset_index(drop=True)

train_nonempty_df = train_df[
    train_df['preprocessed_target_nonzero_voxels'] > 0
].reset_index(drop=True)

print(f'Total manifest subjects with sidecars: {len(model_manifest_df)}')
print(f'Original train subjects: {len(train_df)}')
print(f'Non-empty train subjects for Trial7: {len(train_nonempty_df)}')
print(f'Val subjects: {len(val_df)}')
print(f'Test subjects: {len(test_df)}')
print(
    'Excluded empty train subjects:',
    sorted(set(train_df['subject_id']) - set(train_nonempty_df['subject_id']))
)

REGION_NAMES = [
    'left_anterior',
    'left_middle',
    'left_posterior',
    'right_anterior',
    'right_middle',
    'right_posterior',
]

def make_adjacency():
    edges = [
        (0, 1), (1, 2),
        (3, 4), (4, 5),
        (0, 3), (1, 4), (2, 5),
    ]

    adjacency = torch.eye(NUM_REGIONS, dtype=torch.float32)

    for a, b in edges:
        adjacency[a, b] = 1.0
        adjacency[b, a] = 1.0

    adjacency = adjacency / adjacency.sum(dim=1, keepdim=True)
    return adjacency

ADJACENCY = make_adjacency().to(device)

print('Region names:', REGION_NAMES)
print('Adjacency:', ADJACENCY.detach().cpu())

def make_coarse6_region_map(spatial_shape):
    h, w, d = spatial_shape
    region = np.zeros((h, w, d), dtype=np.int64)

    midline = h // 2
    anterior_end = w // 3
    middle_end = (2 * w) // 3

    region[:midline, :anterior_end, :] = 0
    region[:midline, anterior_end:middle_end, :] = 1
    region[:midline, middle_end:, :] = 2

    region[midline:, :anterior_end, :] = 3
    region[midline:, anterior_end:middle_end, :] = 4
    region[midline:, middle_end:, :] = 5

    return region

def load_regional_features(path):
    arr = np.load(path, allow_pickle=True)

    for key in ['features', 'regional_features', 'node_features', 'region_features']:
        if key in arr.files:
            value = arr[key]
            if value.ndim == 2 and value.shape[0] == NUM_REGIONS:
                return value.astype(np.float32)

    for key in arr.files:
        value = arr[key]
        if value.ndim == 2 and value.shape[0] == NUM_REGIONS:
            return value.astype(np.float32)

    raise ValueError(f'No usable regional feature matrix found in {path}')

example_arr = np.load(train_nonempty_df.iloc[0]['preprocessed_npz_path'], allow_pickle=True)
example_region = make_coarse6_region_map(example_arr['image'].shape[1:])
print('Example image shape:', example_arr['image'].shape)
print('Example region shape:', example_region.shape)
print('Unique regions:', np.unique(example_region))
print('Region map smoke check passed.')

class ISLESTrial7PatchDataset(Dataset):
    def __init__(
        self,
        df,
        regional_feature_path_by_subject,
        patch_size=(128, 128, 64),
        samples_per_epoch=1536,
        positive_prob=0.90,
        seed=20240824,
        balance_by_lesion_bin=True,
    ):
        self.df = df.reset_index(drop=True)
        self.regional_feature_path_by_subject = regional_feature_path_by_subject
        self.patch_size = tuple(patch_size)
        self.samples_per_epoch = samples_per_epoch
        self.positive_prob = positive_prob
        self.balance_by_lesion_bin = balance_by_lesion_bin
        self.rng = np.random.default_rng(seed)

        self.bin_to_indices = {
            lesion_bin: group.index.to_numpy()
            for lesion_bin, group in self.df.groupby('lesion_volume_bin')
        }

        requested_weights = {
            'tiny': 0.22,
            'small': 0.26,
            'medium': 0.26,
            'large': 0.16,
            'very_large': 0.10,
        }

        self.available_bins = [
            b for b in requested_weights
            if b in self.bin_to_indices and len(self.bin_to_indices[b]) > 0
        ]

        raw_weights = np.array([requested_weights[b] for b in self.available_bins], dtype=np.float64)
        self.bin_probs = raw_weights / raw_weights.sum()

    def __len__(self):
        return self.samples_per_epoch

    def _choose_row(self, index):
        if not self.balance_by_lesion_bin:
            return self.df.iloc[index % len(self.df)]

        lesion_bin = self.rng.choice(self.available_bins, p=self.bin_probs)
        case_index = int(self.rng.choice(self.bin_to_indices[lesion_bin]))
        return self.df.iloc[case_index]

    def _pad_to_patch(self, image, target, region):
        _, h, w, d = image.shape
        ph, pw, pd_ = self.patch_size

        pad_h = max(0, ph - h)
        pad_w = max(0, pw - w)
        pad_d = max(0, pd_ - d)

        if pad_h or pad_w or pad_d:
            spatial_pad = (
                (pad_h // 2, pad_h - pad_h // 2),
                (pad_w // 2, pad_w - pad_w // 2),
                (pad_d // 2, pad_d - pad_d // 2),
            )

            image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant', constant_values=0)
            target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant', constant_values=0)
            region = np.pad(region, spatial_pad, mode='edge')

        return image, target, region

    def _sample_patch_start(self, target):
        _, h, w, d = target.shape
        ph, pw, pd_ = self.patch_size

        use_positive = (
            self.rng.random() < self.positive_prob
            and np.count_nonzero(target) > 0
        )

        if use_positive:
            coords = np.argwhere(target[0] > 0)
            center = coords[self.rng.integers(0, len(coords))]
            ch, cw, cd = center
            sh = int(np.clip(ch - ph // 2, 0, h - ph))
            sw = int(np.clip(cw - pw // 2, 0, w - pw))
            sd = int(np.clip(cd - pd_ // 2, 0, d - pd_))
        else:
            sh = int(self.rng.integers(0, h - ph + 1))
            sw = int(self.rng.integers(0, w - pw + 1))
            sd = int(self.rng.integers(0, d - pd_ + 1))

        return sh, sw, sd

    def __getitem__(self, index):
        row = self._choose_row(index)
        subject_id = row['subject_id']

        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.float32)

        region = make_coarse6_region_map(image.shape[1:])
        image, target, region = self._pad_to_patch(image, target, region)

        sh, sw, sd = self._sample_patch_start(target)
        ph, pw, pd_ = self.patch_size

        node_features = load_regional_features(
            self.regional_feature_path_by_subject[subject_id]
        )

        return {
            'image': torch.from_numpy(image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'target': torch.from_numpy(target[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'region': torch.from_numpy(region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]),
            'node_features': torch.from_numpy(node_features),
            'subject_id': subject_id,
        }

class SimpleGraphBlock(nn.Module):
    def __init__(self, dim, dropout=0.10):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, adjacency):
        residual = x
        x = torch.einsum('ij,bjf->bif', adjacency, x)
        x = self.norm(x)
        x = F.gelu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return residual + x

class ConvExpert(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(hidden_channels),
            nn.GELU(),
            nn.Conv3d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(hidden_channels),
            nn.GELU(),
            nn.Conv3d(hidden_channels, 1, kernel_size=1),
        )

    def forward(self, x):
        return self.net(x)

class Trial7CNNInformedGNNRouter(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        node_feature_dim,
        cnn_evidence_dim,
        num_regions,
        adjacency,
        graph_dim=64,
        router_hidden_dim=96,
        num_experts=3,
        adapter_channels=32,
        adapter_init_scale=0.01,
        adapter_max_scale=0.05,
    ):
        super().__init__()

        self.num_regions = num_regions
        self.num_experts = num_experts
        self.adapter_max_scale = adapter_max_scale
        self.register_buffer('adjacency', adjacency.detach().clone())

        self.main_model = SegResNet(
            spatial_dims=3,
            init_filters=16,
            in_channels=in_channels,
            out_channels=out_channels,
            dropout_prob=0.15,
            blocks_down=(1, 2, 2, 4),
            blocks_up=(1, 1, 1),
            norm='INSTANCE',
        )

        self.node_encoder = nn.Sequential(
            nn.LayerNorm(node_feature_dim),
            nn.Linear(node_feature_dim, graph_dim),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(graph_dim, graph_dim),
        )

        self.graph_blocks = nn.ModuleList([
            SimpleGraphBlock(graph_dim, dropout=0.10),
            SimpleGraphBlock(graph_dim, dropout=0.10),
        ])

        router_input_dim = graph_dim + cnn_evidence_dim

        self.router = nn.Sequential(
            nn.LayerNorm(router_input_dim),
            nn.Linear(router_input_dim, router_hidden_dim),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(router_hidden_dim, num_experts),
        )

        self.experts = nn.ModuleList([
            ConvExpert(in_channels, adapter_channels)
            for _ in range(num_experts)
        ])

        init_fraction = float(np.clip(adapter_init_scale / adapter_max_scale, 1e-4, 0.99))
        init_logit = np.log(init_fraction / (1.0 - init_fraction))
        self.adapter_scale_logit = nn.Parameter(torch.tensor(init_logit, dtype=torch.float32))

    def _region_pool_cnn_evidence(self, image, main_logits, region):
        prob = torch.sigmoid(main_logits.detach())

        b, c, h, w, d = image.shape
        total_voxels = float(h * w * d)

        descriptors = []

        for region_idx in range(self.num_regions):
            mask = (region == region_idx).unsqueeze(1)
            mask_f = mask.to(image.dtype)

            denom = mask_f.sum(dim=(2, 3, 4)).clamp_min(1.0)
            volume_fraction = denom / total_voxels

            prob_mean = (prob * mask_f).sum(dim=(2, 3, 4)) / denom

            masked_prob = prob.masked_fill(~mask, -1.0)
            prob_max = masked_prob.amax(dim=(2, 3, 4)).clamp_min(0.0)

            burden_010 = ((prob >= 0.10).to(image.dtype) * mask_f).sum(dim=(2, 3, 4)) / denom
            burden_020 = ((prob >= 0.20).to(image.dtype) * mask_f).sum(dim=(2, 3, 4)) / denom
            burden_030 = ((prob >= 0.30).to(image.dtype) * mask_f).sum(dim=(2, 3, 4)) / denom

            img_mean = (image * mask_f).sum(dim=(2, 3, 4)) / denom
            img_sq_mean = ((image ** 2) * mask_f).sum(dim=(2, 3, 4)) / denom
            img_std = torch.sqrt(torch.clamp(img_sq_mean - img_mean ** 2, min=0.0) + 1e-6)

            descriptor = torch.cat(
                [
                    volume_fraction,
                    prob_mean,
                    prob_max,
                    burden_010,
                    burden_020,
                    burden_030,
                    img_mean,
                    img_std,
                ],
                dim=1,
            )

            descriptors.append(descriptor)

        return torch.stack(descriptors, dim=1)

    def _make_router_maps(self, router_weights, region):
        b, h, w, d = region.shape
        maps = []

        for expert_idx in range(self.num_experts):
            weight_map = torch.zeros(
                (b, 1, h, w, d),
                device=region.device,
                dtype=router_weights.dtype,
            )

            for region_idx in range(self.num_regions):
                mask = (region == region_idx).unsqueeze(1)
                value = router_weights[:, region_idx, expert_idx].view(b, 1, 1, 1, 1)
                weight_map = weight_map + mask.to(router_weights.dtype) * value

            maps.append(weight_map)

        return maps

    def forward(self, image, region, node_features):
        main_logits = self.main_model(image)

        node = self.node_encoder(node_features)

        for block in self.graph_blocks:
            node = block(node, self.adjacency)

        cnn_evidence = self._region_pool_cnn_evidence(image, main_logits, region)
        router_input = torch.cat([node, cnn_evidence], dim=-1)

        router_weights = torch.softmax(self.router(router_input), dim=-1)
        router_maps = self._make_router_maps(router_weights, region)

        expert_logits = [expert(image) for expert in self.experts]
        residual_logits = torch.zeros_like(main_logits)

        for expert_idx, expert_logit in enumerate(expert_logits):
            residual_logits = residual_logits + expert_logit * router_maps[expert_idx]

        adapter_scale = self.adapter_max_scale * torch.sigmoid(self.adapter_scale_logit)
        logits = main_logits + adapter_scale * residual_logits

        return {
            'logits': logits,
            'main_logits': main_logits,
            'residual_logits': residual_logits,
            'router_weights': router_weights,
            'adapter_scale': adapter_scale,
            'cnn_evidence': cnn_evidence,
        }

train_dataset = ISLESTrial7PatchDataset(
    train_nonempty_df,
    regional_feature_path_by_subject=regional_feature_path_by_subject,
    patch_size=PATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_TRAIN_EPOCH,
    positive_prob=POSITIVE_PROB,
    seed=SEED,
    balance_by_lesion_bin=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

model = Trial7CNNInformedGNNRouter(
    in_channels=6,
    out_channels=1,
    node_feature_dim=NODE_FEATURE_DIM,
    cnn_evidence_dim=CNN_EVIDENCE_DIM,
    num_regions=NUM_REGIONS,
    adjacency=ADJACENCY,
    graph_dim=GRAPH_DIM,
    router_hidden_dim=ROUTER_HIDDEN_DIM,
    num_experts=NUM_EXPERTS,
    adapter_channels=ADAPTER_CHANNELS,
    adapter_init_scale=ADAPTER_INIT_SCALE,
    adapter_max_scale=ADAPTER_MAX_SCALE,
).to(device)

loss_fn = DiceFocalLoss(
    sigmoid=True,
    squared_pred=True,
    lambda_dice=1.0,
    lambda_focal=1.0,
    gamma=2.0,
    smooth_nr=1e-5,
    smooth_dr=1e-5,
)

main_params = list(model.main_model.parameters())
adapter_params = [
    p for name, p in model.named_parameters()
    if not name.startswith('main_model.')
]

optimizer = torch.optim.AdamW(
    [
        {'params': main_params, 'lr': MAIN_LR},
        {'params': adapter_params, 'lr': ADAPTER_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=20,
    T_mult=2,
    eta_min=1e-6,
)

try:
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and torch.cuda.is_available())
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and torch.cuda.is_available())

def autocast_context():
    if USE_AMP and torch.cuda.is_available():
        return torch.amp.autocast('cuda')
    return nullcontext()

def set_main_trainable(epoch):
    trainable = epoch >= FREEZE_MAIN_EPOCHS

    for param in model.main_model.parameters():
        param.requires_grad = trainable

    return trainable

def clean_state_dict_keys(state_dict):
    cleaned = {}

    for key, value in state_dict.items():
        new_key = key
        if new_key.startswith('module.'):
            new_key = new_key[len('module.'):]
        if new_key.startswith('main_model.'):
            new_key = new_key[len('main_model.'):]
        cleaned[new_key] = value

    return cleaned

def load_trial2_main_path(model, checkpoint_path):
    if not checkpoint_path.exists():
        print(f'Trial2 checkpoint not found, starting main path from scratch: {checkpoint_path}')
        return

    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state = checkpoint.get('model_state_dict', checkpoint)
    state = clean_state_dict_keys(state)

    model.main_model.load_state_dict(state, strict=True)
    print(f'Loaded Trial2 main path strictly from: {checkpoint_path}')

training_config = {
    'training_run_name': TRAINING_RUN_NAME,
    'preprocessing_version': PREPROCESSING_VERSION,
    'split_name': SPLIT_NAME,
    'trial_note': 'Trial7 fixes Trial6 router by routing from CNN prediction evidence plus GNN regional sidecar context.',
    'trial2_checkpoint_source': str(TRIAL2_BEST_CHECKPOINT_PATH),
    'trial6_failure_note': 'Trial6 router used GNN/sidecar context only and adapter scale grew too high.',
    'patch_size': PATCH_SIZE,
    'samples_per_train_epoch': SAMPLES_PER_TRAIN_EPOCH,
    'batch_size': BATCH_SIZE,
    'positive_prob': POSITIVE_PROB,
    'train_subjects': len(train_nonempty_df),
    'excluded_empty_train_subjects': sorted(
        set(train_df['subject_id']) - set(train_nonempty_df['subject_id'])
    ),
    'max_epochs': MAX_EPOCHS,
    'freeze_main_epochs': FREEZE_MAIN_EPOCHS,
    'min_epochs_before_stopping': MIN_EPOCHS_BEFORE_STOPPING,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'min_delta': MIN_DELTA,
    'main_lr': MAIN_LR,
    'adapter_lr': ADAPTER_LR,
    'adapter_init_scale': ADAPTER_INIT_SCALE,
    'adapter_max_scale': ADAPTER_MAX_SCALE,
    'validation_threshold': VAL_THRESHOLD,
    'loss': 'DiceFocalLoss + auxiliary main-path loss + residual regularization + light router entropy',
    'scheduler': 'CosineAnnealingWarmRestarts',
    'seed': SEED,
}

training_config_path = GENERATED_DIRS['configs'] / f'{TRAINING_RUN_NAME}_training_config.json'
assert_generated_path_is_safe(training_config_path)
training_config_path.write_text(json.dumps(training_config, indent=2), encoding='utf-8')

history = []
best_val_dice = -1.0
start_epoch = 0
epochs_without_improvement = 0

if latest_checkpoint_path.exists():
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if checkpoint.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    if checkpoint.get('scaler_state_dict') is not None:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    history = checkpoint.get('history', [])
    best_val_dice = checkpoint.get('best_val_dice', -1.0)
    epochs_without_improvement = checkpoint.get('epochs_without_improvement', 0)
    start_epoch = checkpoint['epoch'] + 1

    print(f'Resuming Trial7 from: {latest_checkpoint_path}')
    print(f'Start epoch: {start_epoch}')
    print(f'Best val Dice so far: {best_val_dice:.4f}')
    print(f'Epochs without improvement: {epochs_without_improvement}')
else:
    print('Starting new Trial7 run.')
    load_trial2_main_path(model, TRIAL2_BEST_CHECKPOINT_PATH)

set_main_trainable(start_epoch)

batch = next(iter(train_loader))
model.eval()

with torch.no_grad():
    output = model(
        batch['image'].to(device),
        batch['region'].to(device),
        batch['node_features'].to(device),
    )

total_params, trainable_params = count_parameters(model)

print('Batch image:', batch['image'].shape, batch['image'].dtype)
print('Batch target:', batch['target'].shape, batch['target'].dtype)
print('Batch region:', batch['region'].shape, batch['region'].dtype)
print('Batch node features:', batch['node_features'].shape, batch['node_features'].dtype)
print('Batch subject:', batch['subject_id'])
print('Batch target voxels:', int(batch['target'].sum()))
print('Output logits:', output['logits'].shape)
print('Main logits:', output['main_logits'].shape)
print('Residual logits:', output['residual_logits'].shape)
print('CNN evidence:', output['cnn_evidence'].shape)
print('Router weights:', output['router_weights'].shape)
print('Mean router weights:', output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print('Adapter scale:', float(output['adapter_scale'].detach().cpu()))
print(f'Trial7 total parameters: {total_params:,}')
print(f'Trial7 trainable parameters at start epoch: {trainable_params:,}')
print(f'Main path trainable now: {start_epoch >= FREEZE_MAIN_EPOCHS}')
print(f'Validation threshold: {VAL_THRESHOLD}')
print(f'Checkpoints: {checkpoint_dir}')


In [ ]:
def parameter_count_report(model):
    rows = []

    for name, module in model.named_children():
        total = sum(p.numel() for p in module.parameters())
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)

        rows.append({
            'component': name,
            'total_parameters': total,
            'trainable_parameters': trainable,
        })

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    rows.append({
        'component': 'TOTAL',
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
    })

    parameter_df = pd.DataFrame(rows)

    display(parameter_df)

    print(f'Total parameters: {total_params:,}')
    print(f'Trainable parameters: {trainable_params:,}')

    return parameter_df

trial_parameter_df = parameter_count_report(model)


In [ ]:
model.train()
set_main_trainable(start_epoch)

batch = next(iter(train_loader))

images = batch['image'].to(device, non_blocking=True)
targets = batch['target'].to(device, non_blocking=True)
regions = batch['region'].to(device, non_blocking=True)
node_features = batch['node_features'].to(device, non_blocking=True)

optimizer.zero_grad(set_to_none=True)

with autocast_context():
    output = model(images, regions, node_features)

    main_loss = loss_fn(output['main_logits'], targets)
    final_loss = loss_fn(output['logits'], targets)
    residual_reg = output['residual_logits'].pow(2).mean()

    router_weights = output['router_weights']
    router_entropy = -(
        router_weights * torch.log(router_weights.clamp_min(1e-6))
    ).sum(dim=-1).mean()

    loss = (
        final_loss
        + AUX_MAIN_LOSS_WEIGHT * main_loss
        + RESIDUAL_REG_WEIGHT * residual_reg
        - ROUTER_ENTROPY_WEIGHT * router_entropy
    )

if not torch.isfinite(loss):
    raise RuntimeError(f'Non-finite Trial7 preflight loss: {loss.item()}')

scaler.scale(loss).backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    [p for p in model.parameters() if p.requires_grad],
    max_norm=GRAD_CLIP_NORM,
)

scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)

main_grad_params = sum(
    1 for p in model.main_model.parameters()
    if p.grad is not None
)

print('Preflight loss:', float(loss.detach().cpu()))
print('Preflight final loss:', float(final_loss.detach().cpu()))
print('Preflight main loss:', float(main_loss.detach().cpu()))
print('Preflight residual reg:', float(residual_reg.detach().cpu()))
print('Preflight router entropy:', float(router_entropy.detach().cpu()))
print('Preflight grad norm:', float(grad_norm.detach().cpu()))
print('Preflight router:', output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
print('Preflight adapter scale:', float(output['adapter_scale'].detach().cpu()))
print('Main model grad-bearing params during freeze:', main_grad_params)
print('Trial7 preflight passed.')


In [ ]:
def dice_score_binary(pred, target, eps=1e-6):
    pred = pred.astype(bool)
    target = target.astype(bool)

    pred_sum = pred.sum()
    target_sum = target.sum()

    if pred_sum == 0 and target_sum == 0:
        return 1.0

    intersection = np.logical_and(pred, target).sum()
    return float((2.0 * intersection + eps) / (pred_sum + target_sum + eps))

def pad_case_to_roi(image, target, region, roi_size):
    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    pad_h = max(0, ph - h)
    pad_w = max(0, pw - w)
    pad_d = max(0, pd_ - d)

    if not (pad_h or pad_w or pad_d):
        return image, target, region, (h, w, d)

    spatial_pad = (
        (pad_h // 2, pad_h - pad_h // 2),
        (pad_w // 2, pad_w - pad_w // 2),
        (pad_d // 2, pad_d - pad_d // 2),
    )

    image = np.pad(image, ((0, 0),) + spatial_pad, mode='constant', constant_values=0)
    target = np.pad(target, ((0, 0),) + spatial_pad, mode='constant', constant_values=0)
    region = np.pad(region, spatial_pad, mode='edge')

    return image, target, region, (h, w, d)

def get_scan_starts(length, roi, overlap):
    if length <= roi:
        return [0]

    stride = max(1, int(roi * (1.0 - overlap)))
    starts = list(range(0, length - roi + 1, stride))

    if starts[-1] != length - roi:
        starts.append(length - roi)

    return starts

@torch.no_grad()
def trial7_sliding_window_predict(model, image, region, node_features, roi_size, overlap=0.5):
    model.eval()

    _, h, w, d = image.shape
    ph, pw, pd_ = roi_size

    h_starts = get_scan_starts(h, ph, overlap)
    w_starts = get_scan_starts(w, pw, overlap)
    d_starts = get_scan_starts(d, pd_, overlap)

    logit_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32)
    count_sum = torch.zeros((1, 1, h, w, d), dtype=torch.float32)

    router_values = []
    adapter_scales = []

    node_t = torch.from_numpy(node_features[None]).to(device)

    for sh in h_starts:
        for sw in w_starts:
            for sd in d_starts:
                image_patch = image[:, sh:sh+ph, sw:sw+pw, sd:sd+pd_]
                region_patch = region[sh:sh+ph, sw:sw+pw, sd:sd+pd_]

                image_t = torch.from_numpy(image_patch[None]).to(device)
                region_t = torch.from_numpy(region_patch[None]).to(device)

                output = model(image_t, region_t, node_t)
                logits = output['logits'].detach().cpu()

                logit_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += logits
                count_sum[:, :, sh:sh+ph, sw:sw+pw, sd:sd+pd_] += 1.0

                router_values.append(
                    output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy()
                )
                adapter_scales.append(float(output['adapter_scale'].detach().cpu()))

    logits = logit_sum / torch.clamp(count_sum, min=1.0)
    prob = torch.sigmoid(logits)[0, 0].numpy()

    router_mean = np.mean(np.stack(router_values, axis=0), axis=0)
    adapter_scale_mean = float(np.mean(adapter_scales))

    return prob, router_mean, adapter_scale_mean

@torch.no_grad()
def validate_trial7(model, val_df, threshold=VAL_THRESHOLD, max_subjects=None):
    model.eval()

    rows = []
    eval_df = val_df if max_subjects is None else val_df.head(max_subjects)

    for eval_idx, (_, row) in enumerate(eval_df.iterrows(), start=1):
        sid = row['subject_id']

        arr = np.load(row['preprocessed_npz_path'], allow_pickle=True)
        image = arr['image'].astype(np.float32)
        target = arr['target'].astype(np.uint8)
        region = make_coarse6_region_map(image.shape[1:])
        node_features = load_regional_features(regional_feature_path_by_subject[sid])

        image, target, region, original_shape = pad_case_to_roi(
            image,
            target,
            region,
            PATCH_SIZE,
        )

        prob, router_mean, adapter_scale = trial7_sliding_window_predict(
            model,
            image,
            region,
            node_features,
            roi_size=PATCH_SIZE,
            overlap=0.5,
        )

        oh, ow, od = original_shape
        prob = prob[:oh, :ow, :od]
        target_np = target[0, :oh, :ow, :od] > 0

        pred = prob >= threshold
        dice = dice_score_binary(pred, target_np)

        target_voxels = int(target_np.sum())
        pred_voxels = int(pred.sum())

        rows.append({
            'subject_id': sid,
            'dice': dice,
            'target_voxels': target_voxels,
            'pred_voxels': pred_voxels,
            'target_empty': target_voxels == 0,
            'pred_empty': pred_voxels == 0,
            'lesion_volume_bin': row['lesion_volume_bin'],
            'router_expert0': float(router_mean[0]),
            'router_expert1': float(router_mean[1]),
            'router_expert2': float(router_mean[2]),
            'adapter_scale': adapter_scale,
        })

        print(
            f'  Val {eval_idx}/{len(eval_df)} {sid}: '
            f'dice={dice:.4f}, target={target_voxels}, pred={pred_voxels}, '
            f'scale={adapter_scale:.4f}'
        )

    dice_df = pd.DataFrame(rows)
    nonempty_df = dice_df[~dice_df['target_empty']]

    summary = {
        'mean_dice_all': float(dice_df['dice'].mean()) if len(dice_df) else 0.0,
        'mean_dice_nonempty': float(nonempty_df['dice'].mean()) if len(nonempty_df) else 0.0,
        'num_cases': int(len(dice_df)),
        'num_empty_targets': int(dice_df['target_empty'].sum()) if len(dice_df) else 0,
        'mean_pred_voxels': float(dice_df['pred_voxels'].mean()) if len(dice_df) else 0.0,
        'mean_router_expert0': float(dice_df['router_expert0'].mean()) if len(dice_df) else 0.0,
        'mean_router_expert1': float(dice_df['router_expert1'].mean()) if len(dice_df) else 0.0,
        'mean_router_expert2': float(dice_df['router_expert2'].mean()) if len(dice_df) else 0.0,
        'mean_adapter_scale': float(dice_df['adapter_scale'].mean()) if len(dice_df) else 0.0,
    }

    return summary, dice_df

def save_trial7_checkpoint(epoch, best_val_dice, history, epochs_without_improvement, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'best_val_dice': best_val_dice,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'training_config': training_config,
    }

    torch.save(checkpoint, latest_checkpoint_path)

    if is_best:
        torch.save(checkpoint, best_checkpoint_path)

def train_one_epoch_trial7(epoch):
    model.train()

    main_trainable = set_main_trainable(epoch)

    losses = []
    final_losses = []
    main_losses = []
    residual_regs = []
    router_entropies = []
    router_rows = []
    adapter_scales = []

    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        images = batch['image'].to(device, non_blocking=True)
        targets = batch['target'].to(device, non_blocking=True)
        regions = batch['region'].to(device, non_blocking=True)
        node_features = batch['node_features'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast_context():
            output = model(images, regions, node_features)

            main_loss = loss_fn(output['main_logits'], targets)
            final_loss = loss_fn(output['logits'], targets)
            residual_reg = output['residual_logits'].pow(2).mean()

            router_weights = output['router_weights']
            router_entropy = -(
                router_weights * torch.log(router_weights.clamp_min(1e-6))
            ).sum(dim=-1).mean()

            loss = (
                final_loss
                + AUX_MAIN_LOSS_WEIGHT * main_loss
                + RESIDUAL_REG_WEIGHT * residual_reg
                - ROUTER_ENTROPY_WEIGHT * router_entropy
            )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f'Non-finite loss at epoch {epoch}, step {step}: {float(loss.detach().cpu())}'
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=GRAD_CLIP_NORM,
        )

        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.detach().cpu()))
        final_losses.append(float(final_loss.detach().cpu()))
        main_losses.append(float(main_loss.detach().cpu()))
        residual_regs.append(float(residual_reg.detach().cpu()))
        router_entropies.append(float(router_entropy.detach().cpu()))
        router_rows.append(output['router_weights'].mean(dim=(0, 1)).detach().cpu().numpy())
        adapter_scales.append(float(output['adapter_scale'].detach().cpu()))

        if step == 1 or step % 75 == 0 or step == len(train_loader):
            print(
                f'Epoch {epoch:03d} | Step {step:04d}/{len(train_loader)} | '
                f'Loss {losses[-1]:.4f} | Final {final_losses[-1]:.4f} | '
                f'Main {main_losses[-1]:.4f} | Scale {adapter_scales[-1]:.4f} | '
                f'Entropy {router_entropies[-1]:.4f} | '
                f'Main trainable {main_trainable} | '
                f'Grad {float(grad_norm.detach().cpu()):.4f}'
            )

    router_mean = np.mean(np.stack(router_rows, axis=0), axis=0)
    elapsed = time.time() - start_time

    return {
        'main_trainable': main_trainable,
        'train_loss': float(np.mean(losses)),
        'train_final_loss': float(np.mean(final_losses)),
        'train_main_loss': float(np.mean(main_losses)),
        'train_residual_reg': float(np.mean(residual_regs)),
        'train_router_entropy': float(np.mean(router_entropies)),
        'train_router_expert0': float(router_mean[0]),
        'train_router_expert1': float(router_mean[1]),
        'train_router_expert2': float(router_mean[2]),
        'train_adapter_scale': float(np.mean(adapter_scales)),
        'elapsed_seconds': elapsed,
    }

for epoch in range(start_epoch, MAX_EPOCHS):
    print('\n' + '=' * 80)
    print(f'{TRAINING_RUN_NAME} Epoch {epoch}/{MAX_EPOCHS - 1}')
    print('=' * 80)

    train_summary = train_one_epoch_trial7(epoch)

    val_summary = None
    val_df_epoch = None
    val_dice_for_selection = None

    if (epoch + 1) % VAL_EVERY == 0:
        print('\nRunning Trial7 validation...')
        val_summary, val_df_epoch = validate_trial7(
            model,
            val_df,
            threshold=VAL_THRESHOLD,
        )

        val_dice_for_selection = val_summary['mean_dice_nonempty']

        val_detail_path = log_dir / f'val_epoch_{epoch:03d}.csv'
        assert_generated_path_is_safe(val_detail_path)
        val_df_epoch.to_csv(val_detail_path, index=False)

        print(f"Validation Dice, all cases:       {val_summary['mean_dice_all']:.4f}")
        print(f"Validation Dice, non-empty only:  {val_summary['mean_dice_nonempty']:.4f}")
        print(f"Validation empty target cases:    {val_summary['num_empty_targets']}")
        print(f"Validation mean predicted voxels: {val_summary['mean_pred_voxels']:.1f}")
        print(
            'Validation router means: '
            f"{val_summary['mean_router_expert0']:.4f}, "
            f"{val_summary['mean_router_expert1']:.4f}, "
            f"{val_summary['mean_router_expert2']:.4f}"
        )
        print(f"Validation adapter scale:         {val_summary['mean_adapter_scale']:.4f}")
        print(f'Saved validation details: {val_detail_path}')

    if scheduler is not None:
        scheduler.step(epoch + 1)

    current_main_lr = optimizer.param_groups[0]['lr']
    current_adapter_lr = optimizer.param_groups[1]['lr']

    history_row = {
        'epoch': epoch,
        **train_summary,
        'val_dice_all': None if val_summary is None else val_summary['mean_dice_all'],
        'val_dice_nonempty': None if val_summary is None else val_summary['mean_dice_nonempty'],
        'val_empty_targets': None if val_summary is None else val_summary['num_empty_targets'],
        'val_mean_pred_voxels': None if val_summary is None else val_summary['mean_pred_voxels'],
        'val_router_expert0': None if val_summary is None else val_summary['mean_router_expert0'],
        'val_router_expert1': None if val_summary is None else val_summary['mean_router_expert1'],
        'val_router_expert2': None if val_summary is None else val_summary['mean_router_expert2'],
        'val_adapter_scale': None if val_summary is None else val_summary['mean_adapter_scale'],
        'main_lr': current_main_lr,
        'adapter_lr': current_adapter_lr,
    }

    history.append(history_row)

    history_path = log_dir / 'training_history.csv'
    assert_generated_path_is_safe(history_path)
    pd.DataFrame(history).to_csv(history_path, index=False)

    is_best = False

    if val_dice_for_selection is not None:
        if val_dice_for_selection > best_val_dice + MIN_DELTA:
            is_best = True
            best_val_dice = val_dice_for_selection
            epochs_without_improvement = 0
            print(f'New best Trial7 validation Dice, non-empty: {best_val_dice:.4f}')
        else:
            if epoch + 1 >= MIN_EPOCHS_BEFORE_STOPPING:
                epochs_without_improvement += 1
            print(
                f'No validation improvement. '
                f'Early-stop counter: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}'
            )

    save_trial7_checkpoint(
        epoch=epoch,
        best_val_dice=best_val_dice,
        history=history,
        epochs_without_improvement=epochs_without_improvement,
        is_best=is_best,
    )

    print(f'Epoch {epoch} complete.')
    print(f"Train loss: {train_summary['train_loss']:.4f}")
    print(f'Best val Dice, non-empty: {best_val_dice:.4f}')
    print(f'Main LR: {current_main_lr:.8f}')
    print(f'Adapter LR: {current_adapter_lr:.8f}')
    print(f"Elapsed: {train_summary['elapsed_seconds'] / 60:.2f} min")
    print(f'Saved latest checkpoint: {latest_checkpoint_path}')

    can_stop = epoch + 1 >= MIN_EPOCHS_BEFORE_STOPPING

    if can_stop and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f'Early stopping triggered after epoch {epoch}: '
            f'{epochs_without_improvement} epochs without validation improvement, '
            f'with minimum epoch requirement satisfied.'
        )
        break

    if not can_stop:
        print(
            f'Early stopping counter protected until epoch {MIN_EPOCHS_BEFORE_STOPPING - 1}.'
        )

print('Trial7 training complete.')


In [ ]:
!pip -q install nibabel openpyxl

import json
import math
import warnings
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import torch

from google.colab import drive
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)





drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {PROJECT_ROOT}')

GENERATED_DIRS = {
    'metadata': PROJECT_ROOT / 'metadata',
    'splits': PROJECT_ROOT / 'splits',
    'preprocessing': PROJECT_ROOT / 'preprocessing',
    'configs': PROJECT_ROOT / 'configs',
    'checkpoints': PROJECT_ROOT / 'checkpoints',
    'logs': PROJECT_ROOT / 'logs',
    'outputs': PROJECT_ROOT / 'outputs',
}

for path in GENERATED_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

def assert_generated_path_is_safe(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()

    if path != project_root and project_root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')

TRIAL8_AUDIT_NAME = 'v2_vascufate_audit'
AUDIT_DIR = GENERATED_DIRS['preprocessing'] / TRIAL8_AUDIT_NAME

assert_generated_path_is_safe(AUDIT_DIR)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_MANIFEST_PATH = (
    GENERATED_DIRS['metadata'] /
    'model_manifest_v1_trial0.csv'
)

if not MODEL_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Model manifest not found: {MODEL_MANIFEST_PATH}'
    )

manifest_df = pd.read_csv(MODEL_MANIFEST_PATH)

print(f'Project root: {PROJECT_ROOT}')
print(f'Audit directory: {AUDIT_DIR}')
print(f'Manifest: {MODEL_MANIFEST_PATH}')
print(f'Total manifest subjects: {len(manifest_df)}')





SEED = 20240824

def clean_path_value(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    value = str(value).strip()

    if value == '' or value.lower() in {'nan', 'none'}:
        return None

    return Path(value)

def resolve_first_existing_path(row, candidate_columns):
    for column in candidate_columns:
        if column not in row.index:
            continue

        path = clean_path_value(row[column])

        if path is not None and path.exists():
            return path, column

    return None, None

def orientation_string(affine):
    try:
        return ''.join(nib.aff2axcodes(affine))
    except Exception:
        return 'UNKNOWN'

def voxel_sizes_from_affine(affine):
    try:
        return tuple(
            round(float(x), 5)
            for x in nib.affines.voxel_sizes(affine)[:3]
        )
    except Exception:
        return None

def world_center(shape, affine):
    spatial_shape = np.asarray(shape[:3], dtype=np.float64)
    center_index = (spatial_shape - 1.0) / 2.0

    return np.asarray(
        nib.affines.apply_affine(affine, center_index),
        dtype=np.float64,
    )

def sample_volume_array(image, target_samples_per_axis=64):
    shape = image.shape
    slices = []

    for axis, size in enumerate(shape):
        target = target_samples_per_axis if axis < 3 else 8
        step = max(1, int(math.ceil(size / target)))
        slices.append(slice(None, None, step))

    return np.asarray(
        image.dataobj[tuple(slices)],
        dtype=np.float32,
    )

def sampled_intensity_statistics(image):
    sampled = sample_volume_array(image)
    sampled = sampled.reshape(-1)

    finite = sampled[np.isfinite(sampled)]
    nonzero = finite[finite != 0]

    result = {
        'sampled_voxels': int(sampled.size),
        'sampled_finite_voxels': int(finite.size),
        'sampled_nonfinite_voxels': int(sampled.size - finite.size),
        'sampled_zero_fraction': (
            float(np.mean(finite == 0))
            if finite.size
            else np.nan
        ),
    }

    if finite.size:
        quantiles = np.percentile(
            finite,
            [0, 0.5, 1, 25, 50, 75, 99, 99.5, 100],
        )

        result.update({
            'sample_min': float(quantiles[0]),
            'sample_p005': float(quantiles[1]),
            'sample_p01': float(quantiles[2]),
            'sample_p25': float(quantiles[3]),
            'sample_p50': float(quantiles[4]),
            'sample_p75': float(quantiles[5]),
            'sample_p99': float(quantiles[6]),
            'sample_p995': float(quantiles[7]),
            'sample_max': float(quantiles[8]),
        })
    else:
        for name in [
            'sample_min',
            'sample_p005',
            'sample_p01',
            'sample_p25',
            'sample_p50',
            'sample_p75',
            'sample_p99',
            'sample_p995',
            'sample_max',
        ]:
            result[name] = np.nan

    if nonzero.size:
        nonzero_quantiles = np.percentile(
            nonzero,
            [1, 50, 99],
        )

        result.update({
            'nonzero_p01': float(nonzero_quantiles[0]),
            'nonzero_p50': float(nonzero_quantiles[1]),
            'nonzero_p99': float(nonzero_quantiles[2]),
        })
    else:
        result.update({
            'nonzero_p01': np.nan,
            'nonzero_p50': np.nan,
            'nonzero_p99': np.nan,
        })

    return result








MODALITY_SOURCE_COLUMNS = {
    'ncct': [
        'path_raw_ncct',
    ],
    'cta': [
        'path_deriv_cta',
        'path_raw_cta',
    ],
    'cbf': [
        'path_deriv_cbf',
        'path_raw_cbf',
    ],
    'cbv': [
        'path_deriv_cbv',
        'path_raw_cbv',
    ],
    'mtt': [
        'path_deriv_mtt',
        'path_raw_mtt',
    ],
    'tmax': [
        'path_deriv_tmax',
        'path_raw_tmax',
    ],
}

MASK_SOURCE_COLUMNS = {
    'cow_mask': [
        'path_cow_mask',
    ],
    'lvo_mask': [
        'path_lvo_mask',
    ],
    'final_infarct_mask': [
        'path_final_infarct_mask',
    ],
}

RELEVANT_MANIFEST_COLUMNS = [
    'path_raw_ncct',
    'path_raw_cta',
    'path_raw_ctp',
    'path_raw_cbf',
    'path_raw_cbv',
    'path_raw_mtt',
    'path_raw_tmax',
    'path_deriv_cta',
    'path_deriv_ctp',
    'path_deriv_cbf',
    'path_deriv_cbv',
    'path_deriv_mtt',
    'path_deriv_tmax',
    'path_cow_mask',
    'path_lvo_mask',
    'path_final_infarct_mask',
    'path_demographic_baseline_xlsx',
    'path_outcome_xlsx',
    'preprocessed_npz_path',
]





availability_rows = []

for column in RELEVANT_MANIFEST_COLUMNS:
    if column not in manifest_df.columns:
        availability_rows.append({
            'column': column,
            'column_present': False,
            'nonmissing_rows': 0,
            'existing_paths': 0,
            'missing_paths': 0,
            'unique_paths': 0,
        })
        continue

    paths = [
        clean_path_value(value)
        for value in manifest_df[column]
    ]

    paths = [path for path in paths if path is not None]

    existing_paths = sum(path.exists() for path in paths)

    availability_rows.append({
        'column': column,
        'column_present': True,
        'nonmissing_rows': len(paths),
        'existing_paths': existing_paths,
        'missing_paths': len(paths) - existing_paths,
        'unique_paths': len(set(map(str, paths))),
    })

availability_df = pd.DataFrame(availability_rows)

availability_path = AUDIT_DIR / 'manifest_path_availability.csv'
assert_generated_path_is_safe(availability_path)
availability_df.to_csv(availability_path, index=False)

print('\nManifest path availability:')
display(availability_df)





selection_df = manifest_df.copy()

selection_df['dev_split'] = (
    selection_df['dev_split']
    .fillna('unknown')
    .astype(str)
)

selection_df['lesion_volume_bin'] = (
    selection_df['lesion_volume_bin']
    .fillna('unknown')
    .astype(str)
)

stratified_subjects = (
    selection_df
    .groupby(
        ['dev_split', 'lesion_volume_bin'],
        group_keys=False,
    )
    .sample(n=1, random_state=SEED)
)


FOCUS_SUBJECT_IDS = [
    'sub-stroke0001',
    'sub-stroke0004',
    'sub-stroke0057',
    'sub-stroke0070',
    'sub-stroke0094',
    'sub-stroke0114',
    'sub-stroke0141',
    'sub-stroke0162',
    'sub-stroke0171',
]

focus_subjects = selection_df[
    selection_df['subject_id'].isin(FOCUS_SUBJECT_IDS)
]

audit_subject_df = (
    pd.concat(
        [stratified_subjects, focus_subjects],
        ignore_index=True,
    )
    .drop_duplicates(subset='subject_id')
    .sort_values('subject_id')
    .reset_index(drop=True)
)

selected_subjects_path = (
    AUDIT_DIR /
    'selected_audit_subjects.csv'
)

assert_generated_path_is_safe(selected_subjects_path)

audit_subject_df[
    [
        'subject_id',
        'dev_split',
        'lesion_volume_bin',
        'preprocessed_target_nonzero_voxels',
    ]
].to_csv(selected_subjects_path, index=False)

print(f'\nSelected audit subjects: {len(audit_subject_df)}')

display(
    audit_subject_df[
        [
            'subject_id',
            'dev_split',
            'lesion_volume_bin',
            'preprocessed_target_nonzero_voxels',
        ]
    ]
)





npz_schema_rows = []

for _, row in audit_subject_df.head(5).iterrows():
    subject_id = row['subject_id']
    npz_path = clean_path_value(row['preprocessed_npz_path'])

    record = {
        'subject_id': subject_id,
        'npz_path': None if npz_path is None else str(npz_path),
        'readable': False,
        'image_shape': None,
        'target_shape': None,
        'input_columns': None,
        'keys': None,
        'error': None,
    }

    try:
        if npz_path is None or not npz_path.exists():
            raise FileNotFoundError(npz_path)

        with np.load(npz_path, allow_pickle=True) as archive:
            record['keys'] = sorted(archive.files)

            if 'image' in archive:
                record['image_shape'] = tuple(archive['image'].shape)

            if 'target' in archive:
                record['target_shape'] = tuple(archive['target'].shape)

            if 'input_columns' in archive:
                record['input_columns'] = [
                    str(value)
                    for value in archive['input_columns'].tolist()
                ]

        record['readable'] = True

    except Exception as exc:
        record['error'] = f'{type(exc).__name__}: {exc}'

    npz_schema_rows.append(record)

npz_schema_df = pd.DataFrame(npz_schema_rows)

npz_schema_path = AUDIT_DIR / 'existing_npz_schema_audit.csv'
assert_generated_path_is_safe(npz_schema_path)
npz_schema_df.to_csv(npz_schema_path, index=False)

print('\nExisting preprocessed NPZ schema:')
display(npz_schema_df)





intensity_rows = []
source_resolution_rows = []

for subject_index, (_, row) in enumerate(
    audit_subject_df.iterrows(),
    start=1,
):
    subject_id = row['subject_id']

    print(
        f'Intensity audit {subject_index}/'
        f'{len(audit_subject_df)}: {subject_id}'
    )

    for modality, candidates in MODALITY_SOURCE_COLUMNS.items():
        source_path, source_column = resolve_first_existing_path(
            row,
            candidates,
        )

        source_resolution_rows.append({
            'subject_id': subject_id,
            'modality': modality,
            'resolved_source_column': source_column,
            'resolved_path': (
                None if source_path is None else str(source_path)
            ),
            'exists': source_path is not None,
        })

        record = {
            'subject_id': subject_id,
            'dev_split': row['dev_split'],
            'lesion_volume_bin': row['lesion_volume_bin'],
            'modality': modality,
            'source_column': source_column,
            'path': None if source_path is None else str(source_path),
            'readable': False,
            'shape': None,
            'dtype': None,
            'orientation': None,
            'voxel_sizes': None,
            'scl_slope': None,
            'scl_inter': None,
            'file_size_mb': None,
            'error': None,
        }

        try:
            if source_path is None:
                raise FileNotFoundError(
                    f'No existing source among {candidates}'
                )

            image = nib.load(str(source_path))
            slope, intercept = image.header.get_slope_inter()

            record.update({
                'readable': True,
                'shape': tuple(image.shape),
                'dtype': str(image.get_data_dtype()),
                'orientation': orientation_string(image.affine),
                'voxel_sizes': voxel_sizes_from_affine(image.affine),
                'scl_slope': slope,
                'scl_inter': intercept,
                'file_size_mb': (
                    source_path.stat().st_size / (1024 ** 2)
                ),
            })

            record.update(sampled_intensity_statistics(image))

        except Exception as exc:
            record['error'] = f'{type(exc).__name__}: {exc}'

        intensity_rows.append(record)

intensity_df = pd.DataFrame(intensity_rows)
source_resolution_df = pd.DataFrame(source_resolution_rows)

intensity_path = AUDIT_DIR / 'physical_intensity_audit.csv'
source_resolution_path = AUDIT_DIR / 'resolved_modality_sources.csv'

assert_generated_path_is_safe(intensity_path)
assert_generated_path_is_safe(source_resolution_path)

intensity_df.to_csv(intensity_path, index=False)
source_resolution_df.to_csv(source_resolution_path, index=False)

print('\nResolved source columns:')
display(
    source_resolution_df
    .groupby(['modality', 'resolved_source_column'], dropna=False)
    .size()
    .reset_index(name='subjects')
)

print('\nPhysical intensity summary across audited subjects:')

readable_intensity_df = intensity_df[
    intensity_df['readable']
].copy()

if len(readable_intensity_df):
    physical_summary_df = (
        readable_intensity_df
        .groupby('modality')
        .agg(
            audited_subjects=('subject_id', 'nunique'),
            median_sample_min=('sample_min', 'median'),
            median_p01=('sample_p01', 'median'),
            median_p50=('sample_p50', 'median'),
            median_p99=('sample_p99', 'median'),
            median_sample_max=('sample_max', 'median'),
            minimum_seen=('sample_min', 'min'),
            maximum_seen=('sample_max', 'max'),
            median_zero_fraction=('sampled_zero_fraction', 'median'),
            subjects_with_nonfinite=(
                'sampled_nonfinite_voxels',
                lambda values: int((values > 0).sum()),
            ),
        )
        .reset_index()
    )
else:
    physical_summary_df = pd.DataFrame()

physical_summary_path = (
    AUDIT_DIR /
    'physical_intensity_summary.csv'
)

assert_generated_path_is_safe(physical_summary_path)
physical_summary_df.to_csv(physical_summary_path, index=False)

display(physical_summary_df)

print('\nUnreadable modality sources:')

unreadable_intensity_df = intensity_df[
    ~intensity_df['readable']
]

display(
    unreadable_intensity_df[
        [
            'subject_id',
            'modality',
            'source_column',
            'path',
            'error',
        ]
    ]
)





geometry_rows = []

for subject_index, (_, row) in enumerate(
    audit_subject_df.iterrows(),
    start=1,
):
    subject_id = row['subject_id']

    reference_path, reference_column = resolve_first_existing_path(
        row,
        MODALITY_SOURCE_COLUMNS['ncct'],
    )

    if reference_path is None:
        reference_path, reference_column = resolve_first_existing_path(
            row,
            MODALITY_SOURCE_COLUMNS['cta'],
        )

    if reference_path is None:
        geometry_rows.append({
            'subject_id': subject_id,
            'role': 'reference',
            'source_column': None,
            'path': None,
            'readable': False,
            'same_shape_as_reference': False,
            'same_affine_as_reference': False,
            'same_grid_as_reference': False,
            'error': 'No readable NCCT or CTA reference.',
        })
        continue

    try:
        reference_image = nib.load(str(reference_path))
        reference_shape = tuple(reference_image.shape[:3])
        reference_affine = np.asarray(reference_image.affine)
        reference_center = world_center(
            reference_image.shape,
            reference_affine,
        )
    except Exception as exc:
        geometry_rows.append({
            'subject_id': subject_id,
            'role': 'reference',
            'source_column': reference_column,
            'path': str(reference_path),
            'readable': False,
            'same_shape_as_reference': False,
            'same_affine_as_reference': False,
            'same_grid_as_reference': False,
            'error': f'{type(exc).__name__}: {exc}',
        })
        continue

    sources_to_check = {}

    for modality, candidates in MODALITY_SOURCE_COLUMNS.items():
        sources_to_check[modality] = candidates

    for mask_name, candidates in MASK_SOURCE_COLUMNS.items():
        sources_to_check[mask_name] = candidates

    for role, candidates in sources_to_check.items():
        path, source_column = resolve_first_existing_path(
            row,
            candidates,
        )

        record = {
            'subject_id': subject_id,
            'reference_column': reference_column,
            'reference_path': str(reference_path),
            'reference_shape': reference_shape,
            'reference_orientation': orientation_string(
                reference_affine
            ),
            'reference_voxel_sizes': voxel_sizes_from_affine(
                reference_affine
            ),
            'role': role,
            'source_column': source_column,
            'path': None if path is None else str(path),
            'readable': False,
            'shape': None,
            'orientation': None,
            'voxel_sizes': None,
            'same_shape_as_reference': False,
            'same_affine_as_reference': False,
            'same_orientation_as_reference': False,
            'same_grid_as_reference': False,
            'affine_max_abs_difference': None,
            'world_center_distance_mm': None,
            'error': None,
        }

        try:
            if path is None:
                raise FileNotFoundError(
                    f'No existing source among {candidates}'
                )

            image = nib.load(str(path))
            affine = np.asarray(image.affine)
            shape = tuple(image.shape[:3])

            same_shape = shape == reference_shape
            same_affine = np.allclose(
                affine,
                reference_affine,
                rtol=1e-5,
                atol=1e-3,
            )

            orientation = orientation_string(affine)
            reference_orientation = orientation_string(
                reference_affine
            )

            center = world_center(image.shape, affine)
            center_distance = float(
                np.linalg.norm(center - reference_center)
            )

            record.update({
                'readable': True,
                'shape': shape,
                'orientation': orientation,
                'voxel_sizes': voxel_sizes_from_affine(affine),
                'same_shape_as_reference': same_shape,
                'same_affine_as_reference': same_affine,
                'same_orientation_as_reference': (
                    orientation == reference_orientation
                ),
                'same_grid_as_reference': (
                    same_shape and same_affine
                ),
                'affine_max_abs_difference': float(
                    np.max(np.abs(affine - reference_affine))
                ),
                'world_center_distance_mm': center_distance,
            })

        except Exception as exc:
            record['error'] = f'{type(exc).__name__}: {exc}'

        geometry_rows.append(record)

geometry_df = pd.DataFrame(geometry_rows)

geometry_path = AUDIT_DIR / 'geometry_affine_audit.csv'
assert_generated_path_is_safe(geometry_path)
geometry_df.to_csv(geometry_path, index=False)

print('\nGrid-alignment summary:')

geometry_summary_df = (
    geometry_df
    .groupby('role', dropna=False)
    .agg(
        subjects=('subject_id', 'nunique'),
        readable=('readable', 'sum'),
        same_grid=('same_grid_as_reference', 'sum'),
        different_grid=(
            'same_grid_as_reference',
            lambda values: int((~values.astype(bool)).sum()),
        ),
        same_orientation=(
            'same_orientation_as_reference',
            'sum',
        ),
    )
    .reset_index()
)

display(geometry_summary_df)

print('\nSources not already on the reference grid:')

grid_mismatch_df = geometry_df[
    geometry_df['readable']
    & ~geometry_df['same_grid_as_reference']
].copy()

display(
    grid_mismatch_df[
        [
            'subject_id',
            'role',
            'source_column',
            'reference_shape',
            'shape',
            'reference_orientation',
            'orientation',
            'reference_voxel_sizes',
            'voxel_sizes',
            'world_center_distance_mm',
        ]
    ]
)





mask_rows = []

for subject_index, (_, row) in enumerate(
    audit_subject_df.iterrows(),
    start=1,
):
    subject_id = row['subject_id']

    print(
        f'Mask audit {subject_index}/'
        f'{len(audit_subject_df)}: {subject_id}'
    )

    for mask_name, candidates in MASK_SOURCE_COLUMNS.items():
        path, source_column = resolve_first_existing_path(
            row,
            candidates,
        )

        record = {
            'subject_id': subject_id,
            'dev_split': row['dev_split'],
            'lesion_volume_bin': row['lesion_volume_bin'],
            'mask_name': mask_name,
            'source_column': source_column,
            'path': None if path is None else str(path),
            'readable': False,
            'shape': None,
            'orientation': None,
            'voxel_sizes': None,
            'nonzero_voxels': None,
            'nonzero_fraction': None,
            'unique_values': None,
            'binary_mask': None,
            'world_centroid_x': None,
            'world_centroid_y': None,
            'world_centroid_z': None,
            'world_bbox_min': None,
            'world_bbox_max': None,
            'error': None,
        }

        try:
            if path is None:
                raise FileNotFoundError(
                    f'No existing source among {candidates}'
                )

            image = nib.load(str(path))
            array = np.asarray(image.dataobj)
            array = np.squeeze(array)

            if array.ndim != 3:
                raise ValueError(
                    f'Expected 3D mask after squeeze; '
                    f'found shape {array.shape}'
                )

            finite = np.isfinite(array)

            if not finite.all():
                raise ValueError(
                    f'Mask contains '
                    f'{int((~finite).sum())} non-finite voxels.'
                )

            unique_values = np.unique(array)

            if len(unique_values) <= 20:
                unique_values_for_record = [
                    float(value)
                    for value in unique_values.tolist()
                ]
            else:
                unique_values_for_record = (
                    f'{len(unique_values)} unique values; '
                    f'min={float(unique_values.min())}, '
                    f'max={float(unique_values.max())}'
                )

            foreground = array > 0
            nonzero_voxels = int(foreground.sum())

            binary_mask = bool(
                np.all(
                    np.isin(
                        unique_values,
                        [0, 1],
                    )
                )
            )

            record.update({
                'readable': True,
                'shape': tuple(array.shape),
                'orientation': orientation_string(image.affine),
                'voxel_sizes': voxel_sizes_from_affine(
                    image.affine
                ),
                'nonzero_voxels': nonzero_voxels,
                'nonzero_fraction': (
                    float(foreground.mean())
                ),
                'unique_values': unique_values_for_record,
                'binary_mask': binary_mask,
            })

            if nonzero_voxels > 0:
                foreground_indices = np.where(foreground)

                minimum_index = np.array([
                    axis_indices.min()
                    for axis_indices in foreground_indices
                ])

                maximum_index = np.array([
                    axis_indices.max()
                    for axis_indices in foreground_indices
                ])

                centroid_index = np.array([
                    axis_indices.mean()
                    for axis_indices in foreground_indices
                ])

                minimum_world = nib.affines.apply_affine(
                    image.affine,
                    minimum_index,
                )

                maximum_world = nib.affines.apply_affine(
                    image.affine,
                    maximum_index,
                )

                centroid_world = nib.affines.apply_affine(
                    image.affine,
                    centroid_index,
                )

                record.update({
                    'world_centroid_x': float(centroid_world[0]),
                    'world_centroid_y': float(centroid_world[1]),
                    'world_centroid_z': float(centroid_world[2]),
                    'world_bbox_min': tuple(
                        float(value)
                        for value in minimum_world
                    ),
                    'world_bbox_max': tuple(
                        float(value)
                        for value in maximum_world
                    ),
                })

        except Exception as exc:
            record['error'] = f'{type(exc).__name__}: {exc}'

        mask_rows.append(record)

mask_df = pd.DataFrame(mask_rows)

mask_path = AUDIT_DIR / 'cow_lvo_infarct_mask_audit.csv'
assert_generated_path_is_safe(mask_path)
mask_df.to_csv(mask_path, index=False)

print('\nMask audit summary:')

mask_summary_df = (
    mask_df
    .groupby('mask_name', dropna=False)
    .agg(
        subjects=('subject_id', 'nunique'),
        readable=('readable', 'sum'),
        empty_masks=(
            'nonzero_voxels',
            lambda values: int(
                (pd.to_numeric(values, errors='coerce') == 0).sum()
            ),
        ),
        median_nonzero_voxels=('nonzero_voxels', 'median'),
        minimum_nonzero_voxels=('nonzero_voxels', 'min'),
        maximum_nonzero_voxels=('nonzero_voxels', 'max'),
        binary_masks=('binary_mask', 'sum'),
    )
    .reset_index()
)

display(mask_summary_df)

print('\nIndividual CoW and LVO mask records:')

display(
    mask_df[
        mask_df['mask_name'].isin(
            ['cow_mask', 'lvo_mask']
        )
    ][
        [
            'subject_id',
            'mask_name',
            'readable',
            'shape',
            'orientation',
            'voxel_sizes',
            'nonzero_voxels',
            'unique_values',
            'binary_mask',
            'world_centroid_x',
            'world_centroid_y',
            'world_centroid_z',
            'error',
        ]
    ]
)





clinical_schema_rows = []

WORKBOOK_COLUMNS = {
    'demographic_baseline': 'path_demographic_baseline_xlsx',
    'outcome': 'path_outcome_xlsx',
}

for workbook_type, manifest_column in WORKBOOK_COLUMNS.items():
    if manifest_column not in manifest_df.columns:
        continue

    workbook_paths = []

    for value in manifest_df[manifest_column]:
        path = clean_path_value(value)

        if path is not None and path.exists():
            workbook_paths.append(path)

    workbook_paths = sorted(
        set(workbook_paths),
        key=lambda value: str(value),
    )

    print(
        f'\n{workbook_type} unique existing workbooks: '
        f'{len(workbook_paths)}'
    )

    for workbook_path in workbook_paths:
        try:
            workbook = pd.ExcelFile(workbook_path)

            for sheet_name in workbook.sheet_names:
                header_df = pd.read_excel(
                    workbook_path,
                    sheet_name=sheet_name,
                    nrows=0,
                )

                for column_index, column_name in enumerate(
                    header_df.columns
                ):
                    clinical_schema_rows.append({
                        'workbook_type': workbook_type,
                        'workbook_path': str(workbook_path),
                        'sheet_name': sheet_name,
                        'column_index': column_index,
                        'column_name': str(column_name),
                        'error': None,
                    })

        except Exception as exc:
            clinical_schema_rows.append({
                'workbook_type': workbook_type,
                'workbook_path': str(workbook_path),
                'sheet_name': None,
                'column_index': None,
                'column_name': None,
                'error': f'{type(exc).__name__}: {exc}',
            })

clinical_schema_df = pd.DataFrame(clinical_schema_rows)

clinical_schema_path = AUDIT_DIR / 'clinical_workbook_schema.csv'
assert_generated_path_is_safe(clinical_schema_path)
clinical_schema_df.to_csv(clinical_schema_path, index=False)

print('\nClinical workbook schema:')
display(clinical_schema_df)





audit_summary = {
    'trial8_audit_name': TRIAL8_AUDIT_NAME,
    'project_root': str(PROJECT_ROOT),
    'model_manifest_path': str(MODEL_MANIFEST_PATH),
    'total_manifest_subjects': int(len(manifest_df)),
    'audited_subjects': audit_subject_df['subject_id'].tolist(),
    'modality_source_priorities': MODALITY_SOURCE_COLUMNS,
    'mask_source_columns': MASK_SOURCE_COLUMNS,
    'saved_files': {
        'path_availability': str(availability_path),
        'selected_subjects': str(selected_subjects_path),
        'npz_schema': str(npz_schema_path),
        'resolved_sources': str(source_resolution_path),
        'physical_intensities': str(intensity_path),
        'physical_intensity_summary': str(
            physical_summary_path
        ),
        'geometry_affines': str(geometry_path),
        'mask_audit': str(mask_path),
        'clinical_schema': str(clinical_schema_path),
    },
}

audit_summary_path = AUDIT_DIR / 'audit_summary.json'
assert_generated_path_is_safe(audit_summary_path)

audit_summary_path.write_text(
    json.dumps(audit_summary, indent=2),
    encoding='utf-8',
)

print('\n' + '=' * 80)
print('TRIAL8 AUDIT COMPLETE')
print('=' * 80)

print(f'Audit directory: {AUDIT_DIR}')
print(f'Audited subjects: {len(audit_subject_df)}')
print(f'Intensity records: {len(intensity_df)}')
print(f'Geometry records: {len(geometry_df)}')
print(f'Mask records: {len(mask_df)}')
print(f'Clinical schema records: {len(clinical_schema_df)}')

print('\nSaved audit files:')

for name, path in audit_summary['saved_files'].items():
    print(f'  {name}: {path}')

print(f'  audit_summary: {audit_summary_path}')

print('\nNo images or masks were modified.')
print('No Trial8 training has started.')


In [ ]:
!pip -q install nibabel openpyxl scipy

import gc
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)





drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
MANIFEST_PATH = PROJECT_ROOT / 'metadata' / 'model_manifest_v1_trial0.csv'

AUDIT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / 'v2_vascufate_audit'
    / 'phase2_orientation_cow_lvo'
)
QA_DIR = AUDIT_ROOT / 'visual_qa'

AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
QA_DIR.mkdir(parents=True, exist_ok=True)

def assert_project_path(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()

    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')

for path in [AUDIT_ROOT, QA_DIR]:
    assert_project_path(path)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(MANIFEST_PATH)

manifest_df = pd.read_csv(MANIFEST_PATH)

print(f'Project root: {PROJECT_ROOT}')
print(f'Manifest subjects: {len(manifest_df)}')
print(f'Phase 2 audit directory: {AUDIT_ROOT}')





SOURCE_COLUMNS = {
    'ncct': 'path_raw_ncct',
    'cta': 'path_deriv_cta',
    'cbf': 'path_deriv_cbf',
    'cbv': 'path_deriv_cbv',
    'mtt': 'path_deriv_mtt',
    'tmax': 'path_deriv_tmax',
    'cow_mask': 'path_cow_mask',
    'lvo_mask': 'path_lvo_mask',
    'final_infarct_mask': 'path_final_infarct_mask',
}

required_columns = [
    'subject_id',
    'dev_split',
    'lesion_volume_bin',
    *SOURCE_COLUMNS.values(),
    'path_demographic_baseline_xlsx',
    'path_outcome_xlsx',
]

missing_columns = [
    column for column in required_columns
    if column not in manifest_df.columns
]

if missing_columns:
    raise ValueError(f'Missing required manifest columns: {missing_columns}')

def valid_path(value):
    if pd.isna(value):
        return None

    path = Path(str(value))
    return path if path.exists() else None

def orientation_code(image):
    return ''.join(nib.aff2axcodes(image.affine))

def load_canonical_image(path):
    image = nib.load(str(path))
    return nib.as_closest_canonical(image)

def load_canonical_array(path, dtype=np.float32):
    image = load_canonical_image(path)
    array = np.asarray(image.dataobj, dtype=dtype)
    return image, array





orientation_rows = []

for row_number, row in manifest_df.iterrows():
    sid = row['subject_id']
    path = valid_path(row[SOURCE_COLUMNS['ncct']])

    record = {
        'subject_id': sid,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': row['lesion_volume_bin'],
        'path': None if path is None else str(path),
        'readable': False,
        'original_orientation': None,
        'canonical_orientation': None,
        'original_shape': None,
        'canonical_shape': None,
        'original_voxel_sizes': None,
        'canonical_voxel_sizes': None,
        'error': None,
    }

    try:
        if path is None:
            raise FileNotFoundError('NCCT path missing')

        image = nib.load(str(path))
        canonical = nib.as_closest_canonical(image)

        record.update({
            'readable': True,
            'original_orientation': orientation_code(image),
            'canonical_orientation': orientation_code(canonical),
            'original_shape': tuple(int(v) for v in image.shape),
            'canonical_shape': tuple(int(v) for v in canonical.shape),
            'original_voxel_sizes': tuple(
                round(float(v), 5) for v in image.header.get_zooms()[:3]
            ),
            'canonical_voxel_sizes': tuple(
                round(float(v), 5) for v in canonical.header.get_zooms()[:3]
            ),
        })

    except Exception as exc:
        record['error'] = repr(exc)

    orientation_rows.append(record)

orientation_df = pd.DataFrame(orientation_rows)

orientation_path = AUDIT_ROOT / 'orientation_audit_all_subjects.csv'
orientation_df.to_csv(orientation_path, index=False)

print('\nOriginal orientation distribution:')
display(
    orientation_df
    .groupby(['original_orientation', 'canonical_orientation'])
    .size()
    .reset_index(name='subjects')
)

assert orientation_df['readable'].all(), (
    'At least one NCCT was unreadable. Inspect orientation_audit_all_subjects.csv.'
)

assert set(orientation_df['canonical_orientation']) == {'RAS'}, (
    'Canonicalization did not produce RAS orientation for every subject.'
)





representative_ids = []

for original_orientation in sorted(
    orientation_df['original_orientation'].dropna().unique()
):
    candidate = orientation_df[
        orientation_df['original_orientation'] == original_orientation
    ].iloc[0]['subject_id']

    representative_ids.append(candidate)

representative_ids = list(dict.fromkeys(representative_ids))

alignment_rows = []

for sid in representative_ids:
    row = manifest_df[manifest_df['subject_id'] == sid].iloc[0]

    reference_path = valid_path(row[SOURCE_COLUMNS['ncct']])
    reference_original = nib.load(str(reference_path))
    reference_canonical = nib.as_closest_canonical(reference_original)

    for role, column in SOURCE_COLUMNS.items():
        source_path = valid_path(row[column])

        record = {
            'subject_id': sid,
            'role': role,
            'source_column': column,
            'path': None if source_path is None else str(source_path),
            'original_orientation': None,
            'canonical_orientation': None,
            'canonical_shape': None,
            'canonical_shape_match': False,
            'canonical_affine_match': False,
            'canonical_grid_match': False,
            'error': None,
        }

        try:
            if source_path is None:
                raise FileNotFoundError(f'Missing source: {column}')

            source_original = nib.load(str(source_path))
            source_canonical = nib.as_closest_canonical(source_original)

            shape_match = (
                tuple(source_canonical.shape)
                == tuple(reference_canonical.shape)
            )

            affine_match = np.allclose(
                source_canonical.affine,
                reference_canonical.affine,
                atol=1e-4,
                rtol=1e-5,
            )

            record.update({
                'original_orientation': orientation_code(source_original),
                'canonical_orientation': orientation_code(source_canonical),
                'canonical_shape': tuple(
                    int(v) for v in source_canonical.shape
                ),
                'canonical_shape_match': bool(shape_match),
                'canonical_affine_match': bool(affine_match),
                'canonical_grid_match': bool(shape_match and affine_match),
            })

        except Exception as exc:
            record['error'] = repr(exc)

        alignment_rows.append(record)

alignment_df = pd.DataFrame(alignment_rows)

alignment_path = AUDIT_ROOT / 'canonical_alignment_audit.csv'
alignment_df.to_csv(alignment_path, index=False)

print('\nCanonical alignment audit:')
display(alignment_df)

if not alignment_df['canonical_grid_match'].all():
    raise RuntimeError(
        'At least one representative source is misaligned after '
        'canonicalization. Do not continue to preprocessing.'
    )

print('Canonical RAS alignment check passed.')





COW_LABELS = {
    0: 'background',
    1: 'BA',
    2: 'R_PCA',
    3: 'L_PCA',
    4: 'R_ICA',
    5: 'R_MCA',
    6: 'L_ICA',
    7: 'L_MCA',
    8: 'R_Pcom',
    9: 'L_Pcom',
    10: 'Acom',
    11: 'R_ACA',
    12: 'L_ACA',
    13: 'third_A2',
}

cow_mapping_df = pd.DataFrame([
    {'label': label, 'vessel_name': name}
    for label, name in COW_LABELS.items()
])

cow_mapping_path = AUDIT_ROOT / 'cow_label_mapping.csv'
cow_mapping_df.to_csv(cow_mapping_path, index=False)

print('\nCoW label mapping:')
display(cow_mapping_df)


preferred_audit_ids = [
    'sub-stroke0001',
    'sub-stroke0004',
    'sub-stroke0021',
    'sub-stroke0057',
    'sub-stroke0070',
    'sub-stroke0080',
    'sub-stroke0094',
    'sub-stroke0107',
    'sub-stroke0114',
    'sub-stroke0141',
    'sub-stroke0147',
    'sub-stroke0155',
    'sub-stroke0159',
    'sub-stroke0162',
    'sub-stroke0170',
    'sub-stroke0171',
]

audit_subject_df = manifest_df[
    manifest_df['subject_id'].isin(preferred_audit_ids)
    & manifest_df['dev_split'].isin(['train', 'val'])
].copy()

audit_subject_df['_order'] = pd.Categorical(
    audit_subject_df['subject_id'],
    categories=preferred_audit_ids,
    ordered=True,
)

audit_subject_df = (
    audit_subject_df
    .sort_values('_order')
    .drop(columns='_order')
    .reset_index(drop=True)
)

cow_count_rows = []
lvo_association_rows = []

for audit_index, (_, row) in enumerate(
    audit_subject_df.iterrows(),
    start=1,
):
    sid = row['subject_id']

    cow_image, cow = load_canonical_array(
        row['path_cow_mask'],
        dtype=np.float32,
    )
    lvo_image, lvo = load_canonical_array(
        row['path_lvo_mask'],
        dtype=np.float32,
    )

    cow = np.rint(cow).astype(np.int16)
    lvo = lvo > 0

    observed_labels = sorted(
        int(v) for v in np.unique(cow)
    )
    unknown_labels = sorted(
        label for label in observed_labels
        if label not in COW_LABELS
    )

    if unknown_labels:
        raise ValueError(
            f'{sid} contains unknown CoW labels: {unknown_labels}'
        )

    for label in range(1, 14):
        voxel_count = int(np.count_nonzero(cow == label))

        cow_count_rows.append({
            'subject_id': sid,
            'dev_split': row['dev_split'],
            'lesion_volume_bin': row['lesion_volume_bin'],
            'label': label,
            'vessel_name': COW_LABELS[label],
            'voxel_count': voxel_count,
            'present': voxel_count > 0,
        })

    lvo_voxels = int(lvo.sum())
    overlap_labels = sorted(
        int(v)
        for v in np.unique(cow[lvo])
        if int(v) != 0
    )

    nearest_label = None
    nearest_name = None
    nearest_centroid_distance_mm = None

    if lvo_voxels > 0:
        lvo_centroid_voxel = np.argwhere(lvo).mean(axis=0)
        lvo_centroid_world = nib.affines.apply_affine(
            lvo_image.affine,
            lvo_centroid_voxel,
        )

        candidate_distances = []

        for label in range(1, 14):
            vessel_coordinates = np.argwhere(cow == label)

            if len(vessel_coordinates) == 0:
                continue

            vessel_centroid_voxel = vessel_coordinates.mean(axis=0)
            vessel_centroid_world = nib.affines.apply_affine(
                cow_image.affine,
                vessel_centroid_voxel,
            )

            distance_mm = float(
                np.linalg.norm(
                    lvo_centroid_world - vessel_centroid_world
                )
            )

            candidate_distances.append((distance_mm, label))

        if candidate_distances:
            nearest_centroid_distance_mm, nearest_label = min(
                candidate_distances
            )
            nearest_name = COW_LABELS[nearest_label]

    lvo_association_rows.append({
        'subject_id': sid,
        'dev_split': row['dev_split'],
        'lvo_voxels': lvo_voxels,
        'lvo_empty': lvo_voxels == 0,
        'direct_overlap_labels': overlap_labels,
        'direct_overlap_vessels': [
            COW_LABELS[label] for label in overlap_labels
        ],
        'nearest_cow_label': nearest_label,
        'nearest_cow_vessel': nearest_name,
        'centroid_distance_mm': nearest_centroid_distance_mm,
    })

    print(
        f'CoW/LVO audit {audit_index}/{len(audit_subject_df)}: '
        f'{sid}, LVO voxels={lvo_voxels}, '
        f'nearest vessel={nearest_name}'
    )

    del cow_image, cow, lvo_image, lvo
    gc.collect()

cow_count_df = pd.DataFrame(cow_count_rows)
lvo_association_df = pd.DataFrame(lvo_association_rows)

cow_counts_path = AUDIT_ROOT / 'cow_vessel_counts_sample.csv'
lvo_association_path = AUDIT_ROOT / 'lvo_cow_association_sample.csv'

cow_count_df.to_csv(cow_counts_path, index=False)
lvo_association_df.to_csv(lvo_association_path, index=False)

cow_prevalence_df = (
    cow_count_df
    .groupby(['label', 'vessel_name'], as_index=False)
    .agg(
        subjects_present=('present', 'sum'),
        audited_subjects=('subject_id', 'nunique'),
        median_voxels=('voxel_count', 'median'),
        minimum_voxels=('voxel_count', 'min'),
        maximum_voxels=('voxel_count', 'max'),
    )
)

cow_prevalence_df['presence_fraction'] = (
    cow_prevalence_df['subjects_present']
    / cow_prevalence_df['audited_subjects']
)

cow_prevalence_path = AUDIT_ROOT / 'cow_vessel_prevalence_sample.csv'
cow_prevalence_df.to_csv(cow_prevalence_path, index=False)

print('\nCoW vessel prevalence in audited train/validation subjects:')
display(cow_prevalence_df)

print('\nLVO-to-CoW association:')
display(lvo_association_df)





def inspect_workbook(path, category, subject_id):
    path = Path(str(path))
    records = []

    try:
        if path.suffix.lower() == '.csv':
            preview = pd.read_csv(
                path,
                header=None,
                nrows=12,
            )

            records.append({
                'subject_id': subject_id,
                'category': category,
                'path': str(path),
                'sheet_name': 'CSV',
                'preview_rows': int(preview.shape[0]),
                'preview_columns': int(preview.shape[1]),
                'first_12_rows': json.dumps(
                    preview.fillna('').astype(str).values.tolist()
                ),
                'error': None,
            })

        else:
            workbook = pd.ExcelFile(path)

            for sheet_name in workbook.sheet_names:
                preview = pd.read_excel(
                    path,
                    sheet_name=sheet_name,
                    header=None,
                    nrows=12,
                )

                records.append({
                    'subject_id': subject_id,
                    'category': category,
                    'path': str(path),
                    'sheet_name': str(sheet_name),
                    'preview_rows': int(preview.shape[0]),
                    'preview_columns': int(preview.shape[1]),
                    'first_12_rows': json.dumps(
                        preview.fillna('').astype(str).values.tolist()
                    ),
                    'error': None,
                })

    except Exception as exc:
        records.append({
            'subject_id': subject_id,
            'category': category,
            'path': str(path),
            'sheet_name': None,
            'preview_rows': None,
            'preview_columns': None,
            'first_12_rows': None,
            'error': repr(exc),
        })

    return records


clinical_sample_df = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .head(3)
)

clinical_rows = []

for _, row in clinical_sample_df.iterrows():
    clinical_rows.extend(
        inspect_workbook(
            row['path_demographic_baseline_xlsx'],
            category='demographic_baseline',
            subject_id=row['subject_id'],
        )
    )

    clinical_rows.extend(
        inspect_workbook(
            row['path_outcome_xlsx'],
            category='outcome',
            subject_id=row['subject_id'],
        )
    )

clinical_schema_df = pd.DataFrame(clinical_rows)
clinical_schema_path = AUDIT_ROOT / 'clinical_schema_training_sample.csv'
clinical_schema_df.to_csv(clinical_schema_path, index=False)

print('\nClinical workbook schema sample:')
display(
    clinical_schema_df[
        [
            'subject_id',
            'category',
            'sheet_name',
            'preview_rows',
            'preview_columns',
            'error',
        ]
    ]
)

for _, record in clinical_schema_df.iterrows():
    print('\n' + '=' * 80)
    print(
        f"{record['subject_id']} | "
        f"{record['category']} | "
        f"sheet={record['sheet_name']}"
    )

    if pd.notna(record['first_12_rows']):
        preview_values = json.loads(record['first_12_rows'])
        display(pd.DataFrame(preview_values))
    else:
        print(record['error'])





qa_subject_ids = [
    'sub-stroke0001',  
    'sub-stroke0070',  
    'sub-stroke0147',  
    'sub-stroke0159',  
]

qa_subject_ids = [
    sid for sid in qa_subject_ids
    if sid in set(manifest_df['subject_id'])
]

def display_plane(volume, z_index):
    return np.rot90(volume[:, :, z_index])

def add_binary_overlay(ax, mask, z_index, color_map, alpha=0.45):
    overlay = display_plane(mask.astype(np.float32), z_index)
    overlay = np.ma.masked_where(overlay <= 0, overlay)

    ax.imshow(
        overlay,
        cmap=color_map,
        alpha=alpha,
        interpolation='nearest',
    )

qa_summary_rows = []

for sid in qa_subject_ids:
    row = manifest_df[manifest_df['subject_id'] == sid].iloc[0]

    images = {}

    for role, column in SOURCE_COLUMNS.items():
        _, images[role] = load_canonical_array(
            row[column],
            dtype=np.float32,
        )

    target = images['final_infarct_mask'] > 0
    cow = np.rint(images['cow_mask']).astype(np.int16)
    lvo = images['lvo_mask'] > 0

    if target.any():
        z_index = int(
            np.argmax(target.sum(axis=(0, 1)))
        )
        slice_source = 'target'
    elif lvo.any():
        z_index = int(
            np.argmax(lvo.sum(axis=(0, 1)))
        )
        slice_source = 'lvo'
    else:
        z_index = int(
            np.argmax((cow > 0).sum(axis=(0, 1)))
        )
        slice_source = 'cow'

    figure, axes = plt.subplots(
        3,
        3,
        figsize=(15, 14),
        constrained_layout=True,
    )

    display_settings = [
        ('ncct', 'NCCT', 'gray', 0, 100),
        ('cta', 'CTA', 'gray', 0, 300),
        ('cbf', 'CBF', 'magma', 0, 100),
        ('cbv', 'CBV', 'magma', 0, 10),
        ('mtt', 'MTT', 'magma', 0, 20),
        ('tmax', 'Tmax', 'magma', 0, 15),
    ]

    for ax, (role, title, cmap, vmin, vmax) in zip(
        axes.flat[:6],
        display_settings,
    ):
        ax.imshow(
            display_plane(images[role], z_index),
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_title(title)
        ax.axis('off')

    axes[2, 0].imshow(
        display_plane(images['ncct'], z_index),
        cmap='gray',
        vmin=0,
        vmax=100,
    )
    add_binary_overlay(
        axes[2, 0],
        target,
        z_index,
        color_map='Reds',
        alpha=0.50,
    )
    axes[2, 0].set_title('NCCT + final infarct')
    axes[2, 0].axis('off')

    axes[2, 1].imshow(
        display_plane(images['cta'], z_index),
        cmap='gray',
        vmin=0,
        vmax=300,
    )

    cow_overlay = display_plane(cow, z_index)
    cow_overlay = np.ma.masked_where(
        cow_overlay <= 0,
        cow_overlay,
    )

    axes[2, 1].imshow(
        cow_overlay,
        cmap='tab20',
        vmin=1,
        vmax=13,
        alpha=0.70,
        interpolation='nearest',
    )
    axes[2, 1].set_title('CTA + multiclass CoW')
    axes[2, 1].axis('off')

    axes[2, 2].imshow(
        display_plane(images['cta'], z_index),
        cmap='gray',
        vmin=0,
        vmax=300,
    )
    add_binary_overlay(
        axes[2, 2],
        lvo,
        z_index,
        color_map='spring',
        alpha=0.85,
    )
    axes[2, 2].set_title('CTA + LVO')
    axes[2, 2].axis('off')

    original_ncct = nib.load(str(row['path_raw_ncct']))
    original_orientation = orientation_code(original_ncct)

    figure.suptitle(
        f'{sid} | original={original_orientation} | '
        f'canonical=RAS | axial z={z_index} | '
        f'slice selected from {slice_source}',
        fontsize=14,
    )

    qa_path = QA_DIR / f'{sid}_canonical_qa.png'
    figure.savefig(
        qa_path,
        dpi=160,
        bbox_inches='tight',
    )
    plt.show()
    plt.close(figure)

    qa_summary_rows.append({
        'subject_id': sid,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': row['lesion_volume_bin'],
        'original_orientation': original_orientation,
        'canonical_orientation': 'RAS',
        'canonical_shape': tuple(int(v) for v in images['ncct'].shape),
        'display_slice_z': z_index,
        'slice_selection_source': slice_source,
        'target_voxels': int(target.sum()),
        'cow_voxels': int((cow > 0).sum()),
        'lvo_voxels': int(lvo.sum()),
        'qa_path': str(qa_path),
    })

    del images, target, cow, lvo
    gc.collect()

qa_summary_df = pd.DataFrame(qa_summary_rows)
qa_summary_path = AUDIT_ROOT / 'canonical_visual_qa_summary.csv'
qa_summary_df.to_csv(qa_summary_path, index=False)

print('\nCanonical visual QA summary:')
display(qa_summary_df)





phase2_summary = {
    'manifest_subjects': int(len(manifest_df)),
    'original_orientation_counts': {
        str(key): int(value)
        for key, value in (
            orientation_df['original_orientation']
            .value_counts()
            .to_dict()
            .items()
        )
    },
    'canonical_orientations': sorted(
        orientation_df['canonical_orientation']
        .dropna()
        .unique()
        .tolist()
    ),
    'canonical_alignment_representatives': representative_ids,
    'canonical_alignment_passed': bool(
        alignment_df['canonical_grid_match'].all()
    ),
    'cow_label_mapping': {
        str(label): name
        for label, name in COW_LABELS.items()
    },
    'cow_audited_subjects': int(
        cow_count_df['subject_id'].nunique()
    ),
    'lvo_empty_subjects_in_sample': sorted(
        lvo_association_df.loc[
            lvo_association_df['lvo_empty'],
            'subject_id',
        ].tolist()
    ),
    'clinical_subjects_sampled': sorted(
        clinical_sample_df['subject_id'].tolist()
    ),
    'visual_qa_subjects': qa_subject_ids,
    'important_note': (
        'All final Trial8 preprocessing must canonicalize to RAS before '
        'hemisphere comparison, anterior/posterior regional assignment, '
        'or anatomy-aware graph construction.'
    ),
}

phase2_summary_path = AUDIT_ROOT / 'phase2_summary.json'
phase2_summary_path.write_text(
    json.dumps(phase2_summary, indent=2),
    encoding='utf-8',
)

print('\n' + '=' * 80)
print('TRIAL8 PHASE 2 AUDIT COMPLETE')
print('=' * 80)
print(f'Orientation audit:       {orientation_path}')
print(f'Canonical alignment:     {alignment_path}')
print(f'CoW vessel counts:       {cow_counts_path}')
print(f'CoW prevalence:          {cow_prevalence_path}')
print(f'LVO associations:        {lvo_association_path}')
print(f'Clinical schema sample:  {clinical_schema_path}')
print(f'Visual QA directory:     {QA_DIR}')
print(f'Phase 2 summary:         {phase2_summary_path}')
print()
print('No Trial8 training or final preprocessing has occurred yet.')


In [ ]:
!pip -q install nibabel

from pathlib import Path
from itertools import product

import nibabel as nib
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v1_trial0.csv'
)

SUBJECT_ID = 'sub-stroke0141'

SOURCE_COLUMNS = {
    'ncct': 'path_raw_ncct',
    'cta': 'path_deriv_cta',
    'cbf': 'path_deriv_cbf',
    'cbv': 'path_deriv_cbv',
    'mtt': 'path_deriv_mtt',
    'tmax': 'path_deriv_tmax',
    'cow': 'path_cow_mask',
    'lvo': 'path_lvo_mask',
    'target': 'path_final_infarct_mask',
}

manifest_df = pd.read_csv(MANIFEST_PATH)

subject_rows = manifest_df[
    manifest_df['subject_id'] == SUBJECT_ID
]

if len(subject_rows) != 1:
    raise ValueError(
        f'Expected one manifest row for {SUBJECT_ID}; '
        f'found {len(subject_rows)}.'
    )

row = subject_rows.iloc[0]

def orientation_code(image):
    return ''.join(nib.aff2axcodes(image.affine))

def image_corners(shape):
    maximum = np.asarray(shape[:3], dtype=np.float64) - 1.0

    return np.asarray([
        [
            x * maximum[0],
            y * maximum[1],
            z * maximum[2],
        ]
        for x, y, z in product([0, 1], repeat=3)
    ])

def compare_to_reference(source, reference):
    source_corners = image_corners(source.shape)
    reference_corners = image_corners(reference.shape)

    if tuple(source.shape) == tuple(reference.shape):
        source_world = nib.affines.apply_affine(
            source.affine,
            source_corners,
        )
        reference_world = nib.affines.apply_affine(
            reference.affine,
            reference_corners,
        )

        corner_distances = np.linalg.norm(
            source_world - reference_world,
            axis=1,
        )

        maximum_corner_distance = float(
            corner_distances.max()
        )
        mean_corner_distance = float(
            corner_distances.mean()
        )
    else:
        maximum_corner_distance = np.nan
        mean_corner_distance = np.nan

    return {
        'shape_match': (
            tuple(source.shape)
            == tuple(reference.shape)
        ),
        'max_affine_element_difference': float(
            np.max(
                np.abs(
                    source.affine - reference.affine
                )
            )
        ),
        'mean_corner_displacement_mm': (
            mean_corner_distance
        ),
        'maximum_corner_displacement_mm': (
            maximum_corner_distance
        ),
    }

reference_original = nib.load(
    str(row[SOURCE_COLUMNS['ncct']])
)
reference_canonical = nib.as_closest_canonical(
    reference_original
)

diagnostic_rows = []
loaded_images = {}

for role, column in SOURCE_COLUMNS.items():
    original = nib.load(str(row[column]))
    canonical = nib.as_closest_canonical(original)

    loaded_images[role] = {
        'original': original,
        'canonical': canonical,
    }

    original_comparison = compare_to_reference(
        original,
        reference_original,
    )

    canonical_comparison = compare_to_reference(
        canonical,
        reference_canonical,
    )

    _, qform_code = original.get_qform(coded=True)
    _, sform_code = original.get_sform(coded=True)

    diagnostic_rows.append({
        'role': role,
        'source_column': column,
        'original_shape': tuple(original.shape),
        'canonical_shape': tuple(canonical.shape),
        'original_orientation': orientation_code(original),
        'canonical_orientation': orientation_code(canonical),
        'voxel_sizes': tuple(
            round(float(value), 6)
            for value in original.header.get_zooms()[:3]
        ),
        'qform_code': int(qform_code),
        'sform_code': int(sform_code),
        'original_shape_match': (
            original_comparison['shape_match']
        ),
        'original_max_affine_difference': (
            original_comparison[
                'max_affine_element_difference'
            ]
        ),
        'original_max_corner_displacement_mm': (
            original_comparison[
                'maximum_corner_displacement_mm'
            ]
        ),
        'canonical_shape_match': (
            canonical_comparison['shape_match']
        ),
        'canonical_max_affine_difference': (
            canonical_comparison[
                'max_affine_element_difference'
            ]
        ),
        'canonical_mean_corner_displacement_mm': (
            canonical_comparison[
                'mean_corner_displacement_mm'
            ]
        ),
        'canonical_max_corner_displacement_mm': (
            canonical_comparison[
                'maximum_corner_displacement_mm'
            ]
        ),
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)

print(f'Diagnostic subject: {SUBJECT_ID}')
print('\nGrid comparison against canonical NCCT:')
display(diagnostic_df)

print('\nCanonical NCCT affine:')
print(reference_canonical.affine)

print('\nCanonical target affine:')
print(loaded_images['target']['canonical'].affine)

print('\nCanonical affine difference: target minus NCCT')
print(
    loaded_images['target']['canonical'].affine
    - reference_canonical.affine
)

maximum_displacement = diagnostic_df[
    'canonical_max_corner_displacement_mm'
].max()

print(
    '\nMaximum canonical corner displacement across sources: '
    f'{maximum_displacement:.6f} mm'
)

if maximum_displacement <= 0.05:
    print(
        'Interpretation: negligible header rounding. '
        'We can safely harmonize every source to the canonical '
        'NCCT grid during preprocessing.'
    )
elif maximum_displacement <= 0.50:
    print(
        'Interpretation: small grid discrepancy. '
        'Explicit resampling to the canonical NCCT grid is required.'
    )
else:
    print(
        'Interpretation: meaningful spatial discrepancy. '
        'We need visual alignment QA before preprocessing this subject.'
    )


In [ ]:
!pip -q install nibabel scipy openpyxl

import gc
import json
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display
from nibabel.processing import resample_from_to, resample_to_output
from scipy import ndimage





drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v1_trial0.csv'
)

PHASE2_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / 'v2_vascufate_audit'
    / 'phase2_orientation_cow_lvo'
)

ORIENTATION_PATH = (
    PHASE2_ROOT
    / 'orientation_audit_all_subjects.csv'
)

PHASE3_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / 'v2_vascufate_audit'
    / 'phase3_spacing_preservation'
)

QA_DIR = PHASE3_ROOT / 'visual_qa'

PHASE3_ROOT.mkdir(parents=True, exist_ok=True)
QA_DIR.mkdir(parents=True, exist_ok=True)

def assert_project_path(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()

    if path != project_root and project_root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')

for output_path in [PHASE3_ROOT, QA_DIR]:
    assert_project_path(output_path)

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(MANIFEST_PATH)

if not ORIENTATION_PATH.exists():
    raise FileNotFoundError(
        'Phase 2 orientation audit is missing. Run Phase 2 first.'
    )

manifest_df = pd.read_csv(MANIFEST_PATH)
orientation_df = pd.read_csv(ORIENTATION_PATH)

manifest_df = manifest_df.merge(
    orientation_df[
        [
            'subject_id',
            'original_orientation',
            'canonical_orientation',
        ]
    ],
    on='subject_id',
    how='left',
    validate='one_to_one',
)

if not manifest_df['canonical_orientation'].eq('RAS').all():
    raise RuntimeError(
        'Not every subject passed canonical RAS orientation auditing.'
    )

SOURCE_COLUMNS = {
    'ncct': 'path_raw_ncct',
    'cta': 'path_deriv_cta',
    'cbf': 'path_deriv_cbf',
    'cbv': 'path_deriv_cbv',
    'mtt': 'path_deriv_mtt',
    'tmax': 'path_deriv_tmax',
    'cow': 'path_cow_mask',
    'lvo': 'path_lvo_mask',
    'target': 'path_final_infarct_mask',
}

CONTINUOUS_ROLES = {
    'ncct',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
}

MASK_ROLES = {
    'cow',
    'lvo',
    'target',
}

ROLE_CVALS = {
    'ncct': -1000.0,
    'cta': -1000.0,
    'cbf': 0.0,
    'cbv': 0.0,
    'mtt': 0.0,
    'tmax': -30.0,
    'cow': 0.0,
    'lvo': 0.0,
    'target': 0.0,
}

required_columns = [
    'subject_id',
    'dev_split',
    'lesion_volume_bin',
    *SOURCE_COLUMNS.values(),
]

missing_columns = [
    column
    for column in required_columns
    if column not in manifest_df.columns
]

if missing_columns:
    raise ValueError(
        f'Missing required manifest columns: {missing_columns}'
    )

CANDIDATE_SPACINGS = {
    'detail_0p8x0p8x2': (0.8, 0.8, 2.0),
    'balanced_1x1x2': (1.0, 1.0, 2.0),
    'efficient_1x1x3': (1.0, 1.0, 3.0),
}

EXPECTED_MODEL_CHANNELS = 12
GRID_ATOL = 1e-4
GRID_RTOL = 1e-5
LARGE_DISPLACEMENT_WARNING_MM = 2.0

print(f'Manifest subjects: {len(manifest_df)}')
print(f'Phase 3 output: {PHASE3_ROOT}')
print(f'Candidate spacings: {CANDIDATE_SPACINGS}')





def orientation_code(image):
    return ''.join(nib.aff2axcodes(image.affine))

def canonical_image(path):
    image = nib.load(str(path))
    canonical = nib.as_closest_canonical(image)

    if orientation_code(canonical) != 'RAS':
        raise RuntimeError(
            f'Canonicalization failed for: {path}'
        )

    return canonical

def voxel_volume_mm3(image):
    return float(
        abs(np.linalg.det(image.affine[:3, :3]))
    )

def grids_match(first, second):
    return (
        tuple(first.shape) == tuple(second.shape)
        and np.allclose(
            first.affine,
            second.affine,
            atol=GRID_ATOL,
            rtol=GRID_RTOL,
        )
    )

def image_corners(shape):
    maximum = np.asarray(
        shape[:3],
        dtype=np.float64,
    ) - 1.0

    return np.asarray([
        [
            x * maximum[0],
            y * maximum[1],
            z * maximum[2],
        ]
        for x, y, z in product([0, 1], repeat=3)
    ])

def grid_comparison(source, reference):
    shape_match = (
        tuple(source.shape)
        == tuple(reference.shape)
    )

    affine_difference = float(
        np.max(
            np.abs(
                source.affine - reference.affine
            )
        )
    )

    if shape_match:
        corners = image_corners(reference.shape)

        source_world = nib.affines.apply_affine(
            source.affine,
            corners,
        )

        reference_world = nib.affines.apply_affine(
            reference.affine,
            corners,
        )

        distances = np.linalg.norm(
            source_world - reference_world,
            axis=1,
        )

        mean_corner_displacement = float(
            distances.mean()
        )
        maximum_corner_displacement = float(
            distances.max()
        )
    else:
        mean_corner_displacement = np.nan
        maximum_corner_displacement = np.nan

    return {
        'shape_match': bool(shape_match),
        'affine_match': bool(
            np.allclose(
                source.affine,
                reference.affine,
                atol=GRID_ATOL,
                rtol=GRID_RTOL,
            )
        ),
        'grid_match': bool(
            grids_match(source, reference)
        ),
        'maximum_affine_element_difference': (
            affine_difference
        ),
        'mean_corner_displacement_mm': (
            mean_corner_displacement
        ),
        'maximum_corner_displacement_mm': (
            maximum_corner_displacement
        ),
    }

def image_to_array(image, dtype=np.float32):
    return np.asarray(
        image.dataobj,
        dtype=dtype,
    )

def harmonize_to_reference(
    source_image,
    reference_image,
    order,
    cval,
):
    comparison = grid_comparison(
        source_image,
        reference_image,
    )

    if comparison['grid_match']:
        output_image = source_image
        was_resampled = False
    else:
        output_image = resample_from_to(
            source_image,
            (
                reference_image.shape,
                reference_image.affine,
            ),
            order=order,
            mode='constant',
            cval=float(cval),
        )
        was_resampled = True

    output_comparison = grid_comparison(
        output_image,
        reference_image,
    )

    if not output_comparison['grid_match']:
        raise RuntimeError(
            'Source did not align to the NCCT reference grid '
            'after explicit harmonization.'
        )

    return output_image, was_resampled, comparison

def load_and_harmonize_subject(row):
    subject_id = row['subject_id']

    canonical_images = {}

    for role, column in SOURCE_COLUMNS.items():
        source_path = Path(str(row[column]))

        if not source_path.exists():
            raise FileNotFoundError(
                f'{subject_id} missing {role}: {source_path}'
            )

        canonical_images[role] = canonical_image(
            source_path
        )

    reference_image = canonical_images['ncct']

    harmonized_arrays = {}
    audit_rows = []

    for role, source_image in canonical_images.items():
        if role in CONTINUOUS_ROLES:
            interpolation_order = 1
        elif role in MASK_ROLES:
            interpolation_order = 0
        else:
            raise ValueError(f'Unknown role: {role}')

        if role == 'ncct':
            harmonized_image = reference_image
            was_resampled = False
            comparison = grid_comparison(
                source_image,
                reference_image,
            )
        else:
            (
                harmonized_image,
                was_resampled,
                comparison,
            ) = harmonize_to_reference(
                source_image=source_image,
                reference_image=reference_image,
                order=interpolation_order,
                cval=ROLE_CVALS[role],
            )

        harmonized_arrays[role] = image_to_array(
            harmonized_image,
            dtype=np.float32,
        )

        maximum_displacement = comparison[
            'maximum_corner_displacement_mm'
        ]

        large_displacement_warning = bool(
            np.isfinite(maximum_displacement)
            and maximum_displacement
            > LARGE_DISPLACEMENT_WARNING_MM
        )

        audit_rows.append({
            'subject_id': subject_id,
            'role': role,
            'source_column': SOURCE_COLUMNS[role],
            'canonical_orientation': (
                orientation_code(source_image)
            ),
            'source_shape': tuple(
                int(value)
                for value in source_image.shape
            ),
            'reference_shape': tuple(
                int(value)
                for value in reference_image.shape
            ),
            'source_spacing': tuple(
                round(float(value), 6)
                for value
                in source_image.header.get_zooms()[:3]
            ),
            'reference_spacing': tuple(
                round(float(value), 6)
                for value
                in reference_image.header.get_zooms()[:3]
            ),
            'initial_shape_match': comparison[
                'shape_match'
            ],
            'initial_affine_match': comparison[
                'affine_match'
            ],
            'initial_grid_match': comparison[
                'grid_match'
            ],
            'maximum_affine_element_difference': (
                comparison[
                    'maximum_affine_element_difference'
                ]
            ),
            'mean_corner_displacement_mm': (
                comparison[
                    'mean_corner_displacement_mm'
                ]
            ),
            'maximum_corner_displacement_mm': (
                maximum_displacement
            ),
            'interpolation_order': interpolation_order,
            'was_resampled_to_ncct': was_resampled,
            'large_displacement_warning': (
                large_displacement_warning
            ),
            'final_grid_match': True,
        })

    return (
        reference_image,
        canonical_images,
        harmonized_arrays,
        audit_rows,
    )





def binary_dice(first, second, eps=1e-6):
    first = np.asarray(first, dtype=bool)
    second = np.asarray(second, dtype=bool)

    first_sum = int(first.sum())
    second_sum = int(second.sum())

    if first_sum == 0 and second_sum == 0:
        return 1.0

    intersection = int(
        np.logical_and(first, second).sum()
    )

    return float(
        (2.0 * intersection + eps)
        / (first_sum + second_sum + eps)
    )

def coverage_fraction(mask, support):
    mask = np.asarray(mask, dtype=bool)
    support = np.asarray(support, dtype=bool)

    mask_voxels = int(mask.sum())

    if mask_voxels == 0:
        return np.nan

    return float(
        np.logical_and(mask, support).sum()
        / mask_voxels
    )

def largest_component(mask):
    mask = np.asarray(mask, dtype=bool)

    labels, component_count = ndimage.label(
        mask,
        structure=ndimage.generate_binary_structure(
            3,
            1,
        ),
    )

    if component_count == 0:
        return np.zeros_like(mask, dtype=bool)

    component_sizes = np.bincount(
        labels.ravel()
    )
    component_sizes[0] = 0

    largest_label = int(
        component_sizes.argmax()
    )

    return labels == largest_label

def construct_provisional_support(
    ncct,
    cbf,
    cbv,
    mtt,
    tmax,
):
    finite = (
        np.isfinite(ncct)
        & np.isfinite(cbf)
        & np.isfinite(cbv)
        & np.isfinite(mtt)
        & np.isfinite(tmax)
    )

    ct_soft_tissue = (
        (ncct > -100.0)
        & (ncct < 200.0)
    )

    perfusion_defined = (
        (tmax > -29.5)
        | (cbf > 0.0)
        | (cbv > 0.0)
        | (mtt > 0.0)
    )

    support = (
        finite
        & ct_soft_tissue
        & perfusion_defined
    )

    structure = (
        ndimage.generate_binary_structure(3, 1)
    )

    support = ndimage.binary_closing(
        support,
        structure=structure,
        iterations=2,
    )

    for z_index in range(support.shape[2]):
        support[:, :, z_index] = (
            ndimage.binary_fill_holes(
                support[:, :, z_index]
            )
        )

    support = largest_component(support)

    support = ndimage.binary_dilation(
        support,
        structure=structure,
        iterations=1,
    )

    return support.astype(bool)

def resample_array_to_grid(
    array,
    source_affine,
    target_grid,
    order,
    cval=0.0,
):
    source_image = nib.Nifti1Image(
        np.asarray(array, dtype=np.float32),
        source_affine,
    )

    result = resample_from_to(
        source_image,
        target_grid,
        order=order,
        mode='constant',
        cval=float(cval),
    )

    return np.asarray(
        result.dataobj,
        dtype=np.float32,
    )

def resample_binary_back_to_native(
    mask,
    resampled_affine,
    native_grid,
):
    mask_image = nib.Nifti1Image(
        np.asarray(mask, dtype=np.uint8),
        resampled_affine,
    )

    restored = resample_from_to(
        mask_image,
        native_grid,
        order=0,
        mode='constant',
        cval=0.0,
    )

    return np.asarray(
        restored.dataobj
    ) > 0

def choose_display_slice(
    mask,
    fallback_mask=None,
):
    mask = np.asarray(mask, dtype=bool)

    if mask.any():
        return int(
            np.argmax(
                mask.sum(axis=(0, 1))
            )
        )

    if (
        fallback_mask is not None
        and np.asarray(fallback_mask).any()
    ):
        fallback_mask = np.asarray(
            fallback_mask,
            dtype=bool,
        )

        return int(
            np.argmax(
                fallback_mask.sum(axis=(0, 1))
            )
        )

    return int(mask.shape[2] // 2)

def display_plane(volume, z_index):
    return np.rot90(
        np.asarray(volume)[:, :, z_index]
    )

def add_overlay(
    axis,
    mask,
    z_index,
    cmap,
    alpha=0.5,
):
    overlay = display_plane(
        mask,
        z_index,
    )

    overlay = np.ma.masked_where(
        overlay <= 0,
        overlay,
    )

    axis.imshow(
        overlay,
        cmap=cmap,
        alpha=alpha,
        interpolation='nearest',
    )





spacing_rows = []

for _, row in manifest_df.iterrows():
    subject_id = row['subject_id']

    try:
        image = nib.load(
            str(row[SOURCE_COLUMNS['ncct']])
        )

        spacing = image.header.get_zooms()[:3]

        spacing_rows.append({
            'subject_id': subject_id,
            'original_orientation': (
                orientation_code(image)
            ),
            'spacing_x': float(spacing[0]),
            'spacing_y': float(spacing[1]),
            'spacing_z': float(spacing[2]),
            'readable': True,
            'error': None,
        })

    except Exception as exc:
        spacing_rows.append({
            'subject_id': subject_id,
            'original_orientation': None,
            'spacing_x': None,
            'spacing_y': None,
            'spacing_z': None,
            'readable': False,
            'error': repr(exc),
        })

spacing_df = pd.DataFrame(spacing_rows)

spacing_path = (
    PHASE3_ROOT
    / 'native_spacing_all_subjects.csv'
)

spacing_df.to_csv(
    spacing_path,
    index=False,
)

if not spacing_df['readable'].all():
    raise RuntimeError(
        'At least one NCCT header was unreadable.'
    )

print('\nNative voxel-spacing summary:')
display(
    spacing_df[
        [
            'spacing_x',
            'spacing_y',
            'spacing_z',
        ]
    ].describe()
)

print('\nRounded through-plane spacing distribution:')
display(
    spacing_df
    .assign(
        spacing_z_rounded=(
            spacing_df['spacing_z'].round(2)
        )
    )
    .groupby('spacing_z_rounded')
    .size()
    .reset_index(name='subjects')
)





eligible_df = manifest_df[
    manifest_df['dev_split'].isin(
        ['train', 'val']
    )
].copy()

lesion_bin_order = [
    'empty',
    'tiny',
    'small',
    'medium',
    'large',
    'very_large',
]

selected_ids = []

for lesion_bin in lesion_bin_order:
    bin_df = eligible_df[
        eligible_df['lesion_volume_bin']
        == lesion_bin
    ]

    for orientation in ['LAS', 'LPS']:
        candidate_df = (
            bin_df[
                bin_df['original_orientation']
                == orientation
            ]
            .sort_values('subject_id')
        )

        if len(candidate_df):
            selected_ids.append(
                candidate_df.iloc[0][
                    'subject_id'
                ]
            )

if 'sub-stroke0171' in set(
    eligible_df['subject_id']
):
    selected_ids.append(
        'sub-stroke0171'
    )

selected_ids = list(
    dict.fromkeys(selected_ids)
)

selected_df = eligible_df[
    eligible_df['subject_id'].isin(
        selected_ids
    )
].copy()

selected_df['_selection_order'] = (
    pd.Categorical(
        selected_df['subject_id'],
        categories=selected_ids,
        ordered=True,
    )
)

selected_df = (
    selected_df
    .sort_values('_selection_order')
    .drop(columns='_selection_order')
    .reset_index(drop=True)
)

selected_path = (
    PHASE3_ROOT
    / 'selected_spacing_audit_subjects.csv'
)

selected_df.to_csv(
    selected_path,
    index=False,
)

print(f'\nSelected subjects: {len(selected_df)}')

display_columns = [
    'subject_id',
    'dev_split',
    'lesion_volume_bin',
    'original_orientation',
]

if (
    'preprocessed_target_nonzero_voxels'
    in selected_df.columns
):
    display_columns.append(
        'preprocessed_target_nonzero_voxels'
    )

display(selected_df[display_columns])





harmonization_rows = []
native_rows = []
spacing_result_rows = []

qa_preferred_ids = [
    'sub-stroke0001',
    'sub-stroke0070',
    'sub-stroke0147',
    'sub-stroke0159',
    'sub-stroke0171',
]

qa_ids = [
    subject_id
    for subject_id in qa_preferred_ids
    if subject_id
    in set(selected_df['subject_id'])
]

for subject_index, (_, row) in enumerate(
    selected_df.iterrows(),
    start=1,
):
    subject_id = row['subject_id']

    print(
        f'\nProcessing '
        f'{subject_index}/{len(selected_df)}: '
        f'{subject_id}'
    )

    (
        reference_image,
        source_images,
        arrays,
        subject_harmonization_rows,
    ) = load_and_harmonize_subject(row)

    harmonization_rows.extend(
        subject_harmonization_rows
    )

    warned_roles = [
        audit_row['role']
        for audit_row
        in subject_harmonization_rows
        if audit_row[
            'large_displacement_warning'
        ]
    ]

    if warned_roles:
        print(
            '  Large displacement warning for: '
            f'{warned_roles}'
        )

    resampled_roles = [
        audit_row['role']
        for audit_row
        in subject_harmonization_rows
        if audit_row[
            'was_resampled_to_ncct'
        ]
    ]

    if resampled_roles:
        print(
            '  Harmonized to NCCT grid: '
            f'{resampled_roles}'
        )
    else:
        print(
            '  All canonical sources already '
            'matched the NCCT grid.'
        )

    ncct = arrays['ncct']
    cbf = arrays['cbf']
    cbv = arrays['cbv']
    mtt = arrays['mtt']
    tmax = arrays['tmax']

    target_harmonized = (
        arrays['target'] > 0.5
    )

    cow_harmonized = np.rint(
        arrays['cow']
    ).astype(np.int16)

    cow_union_harmonized = (
        cow_harmonized > 0
    )

    lvo_harmonized = (
        arrays['lvo'] > 0.5
    )

    target_source = (
        image_to_array(
            source_images['target']
        ) > 0
    )

    lvo_source = (
        image_to_array(
            source_images['lvo']
        ) > 0
    )

    cow_source = np.rint(
        image_to_array(
            source_images['cow']
        )
    ).astype(np.int16)

    cow_union_source = (
        cow_source > 0
    )

    support_harmonized = (
        construct_provisional_support(
            ncct=ncct,
            cbf=cbf,
            cbv=cbv,
            mtt=mtt,
            tmax=tmax,
        )
    )

    reference_voxel_volume = (
        voxel_volume_mm3(reference_image)
    )

    source_target_voxel_volume = (
        voxel_volume_mm3(
            source_images['target']
        )
    )

    source_lvo_voxel_volume = (
        voxel_volume_mm3(
            source_images['lvo']
        )
    )

    source_cow_voxel_volume = (
        voxel_volume_mm3(
            source_images['cow']
        )
    )

    source_target_voxels = int(
        target_source.sum()
    )

    harmonized_target_voxels = int(
        target_harmonized.sum()
    )

    source_target_volume_ml = (
        source_target_voxels
        * source_target_voxel_volume
        / 1000.0
    )

    harmonized_target_volume_ml = (
        harmonized_target_voxels
        * reference_voxel_volume
        / 1000.0
    )

    source_lvo_voxels = int(
        lvo_source.sum()
    )

    harmonized_lvo_voxels = int(
        lvo_harmonized.sum()
    )

    source_lvo_volume_ml = (
        source_lvo_voxels
        * source_lvo_voxel_volume
        / 1000.0
    )

    harmonized_lvo_volume_ml = (
        harmonized_lvo_voxels
        * reference_voxel_volume
        / 1000.0
    )

    source_cow_voxels = int(
        cow_union_source.sum()
    )

    harmonized_cow_voxels = int(
        cow_union_harmonized.sum()
    )

    source_cow_volume_ml = (
        source_cow_voxels
        * source_cow_voxel_volume
        / 1000.0
    )

    harmonized_cow_volume_ml = (
        harmonized_cow_voxels
        * reference_voxel_volume
        / 1000.0
    )

    support_coordinates = np.argwhere(
        support_harmonized
    )

    if len(support_coordinates):
        support_minimum = (
            support_coordinates.min(axis=0)
        )
        support_maximum = (
            support_coordinates.max(axis=0)
        )

        support_midpoint = (
            support_minimum + support_maximum
        ) / 2.0

        support_midpoint_world = (
            nib.affines.apply_affine(
                reference_image.affine,
                support_midpoint,
            )
        )

        estimated_midline_world_x = float(
            support_midpoint_world[0]
        )
    else:
        estimated_midline_world_x = np.nan

    native_rows.append({
        'subject_id': subject_id,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': (
            row['lesion_volume_bin']
        ),
        'original_orientation': (
            row['original_orientation']
        ),
        'canonical_orientation': 'RAS',
        'reference_shape': tuple(
            int(value)
            for value in reference_image.shape
        ),
        'reference_spacing': tuple(
            round(float(value), 6)
            for value
            in reference_image.header.get_zooms()[:3]
        ),
        'reference_voxel_volume_mm3': (
            reference_voxel_volume
        ),
        'source_target_voxels': (
            source_target_voxels
        ),
        'harmonized_target_voxels': (
            harmonized_target_voxels
        ),
        'source_target_volume_ml': (
            source_target_volume_ml
        ),
        'harmonized_target_volume_ml': (
            harmonized_target_volume_ml
        ),
        'harmonization_target_volume_error_ml': (
            harmonized_target_volume_ml
            - source_target_volume_ml
        ),
        'harmonization_target_absolute_volume_error_ml': abs(
            harmonized_target_volume_ml
            - source_target_volume_ml
        ),
        'source_lvo_voxels': (
            source_lvo_voxels
        ),
        'harmonized_lvo_voxels': (
            harmonized_lvo_voxels
        ),
        'source_lvo_volume_ml': (
            source_lvo_volume_ml
        ),
        'harmonized_lvo_volume_ml': (
            harmonized_lvo_volume_ml
        ),
        'source_cow_voxels': (
            source_cow_voxels
        ),
        'harmonized_cow_voxels': (
            harmonized_cow_voxels
        ),
        'source_cow_volume_ml': (
            source_cow_volume_ml
        ),
        'harmonized_cow_volume_ml': (
            harmonized_cow_volume_ml
        ),
        'support_voxels': int(
            support_harmonized.sum()
        ),
        'support_fraction_of_image': float(
            support_harmonized.mean()
        ),
        'target_coverage_by_support': (
            coverage_fraction(
                target_harmonized,
                support_harmonized,
            )
        ),
        'cow_coverage_by_support': (
            coverage_fraction(
                cow_union_harmonized,
                support_harmonized,
            )
        ),
        'lvo_coverage_by_support': (
            coverage_fraction(
                lvo_harmonized,
                support_harmonized,
            )
        ),
        'estimated_midline_world_x_mm': (
            estimated_midline_world_x
        ),
        'harmonized_roles': (
            resampled_roles
        ),
        'large_displacement_warning_roles': (
            warned_roles
        ),
    })

    native_grid = (
        reference_image.shape,
        reference_image.affine,
    )

    candidate_snapshots = {}

    for (
        candidate_name,
        candidate_spacing,
    ) in CANDIDATE_SPACINGS.items():
        reference_resampled = (
            resample_to_output(
                reference_image,
                voxel_sizes=candidate_spacing,
                order=1,
                mode='constant',
                cval=-1000.0,
            )
        )

        output_grid = (
            reference_resampled.shape,
            reference_resampled.affine,
        )

        reference_ncct = image_to_array(
            reference_resampled
        )

        target_soft = (
            resample_array_to_grid(
                target_harmonized.astype(
                    np.float32
                ),
                reference_image.affine,
                output_grid,
                order=1,
                cval=0.0,
            )
        )

        target_soft = np.clip(
            target_soft,
            0.0,
            1.0,
        )

        target_nearest = (
            resample_array_to_grid(
                target_harmonized.astype(
                    np.float32
                ),
                reference_image.affine,
                output_grid,
                order=0,
                cval=0.0,
            ) > 0.5
        )

        target_soft_01 = (
            target_soft >= 0.10
        )

        target_soft_05 = (
            target_soft >= 0.50
        )

        lvo_soft = resample_array_to_grid(
            lvo_harmonized.astype(
                np.float32
            ),
            reference_image.affine,
            output_grid,
            order=1,
            cval=0.0,
        )

        lvo_soft = np.clip(
            lvo_soft,
            0.0,
            1.0,
        )

        lvo_nearest = (
            resample_array_to_grid(
                lvo_harmonized.astype(
                    np.float32
                ),
                reference_image.affine,
                output_grid,
                order=0,
                cval=0.0,
            ) > 0.5
        )

        lvo_soft_01 = (
            lvo_soft >= 0.10
        )

        cow_soft = resample_array_to_grid(
            cow_union_harmonized.astype(
                np.float32
            ),
            reference_image.affine,
            output_grid,
            order=1,
            cval=0.0,
        )

        cow_soft = np.clip(
            cow_soft,
            0.0,
            1.0,
        )

        cow_soft_01 = (
            cow_soft >= 0.10
        )

        support_resampled = (
            resample_array_to_grid(
                support_harmonized.astype(
                    np.float32
                ),
                reference_image.affine,
                output_grid,
                order=0,
                cval=0.0,
            ) > 0.5
        )

        nearest_back = (
            resample_binary_back_to_native(
                target_nearest,
                reference_resampled.affine,
                native_grid,
            )
        )

        soft01_back = (
            resample_binary_back_to_native(
                target_soft_01,
                reference_resampled.affine,
                native_grid,
            )
        )

        soft05_back = (
            resample_binary_back_to_native(
                target_soft_05,
                reference_resampled.affine,
                native_grid,
            )
        )

        output_voxel_volume = (
            voxel_volume_mm3(
                reference_resampled
            )
        )

        nearest_volume_ml = (
            int(target_nearest.sum())
            * output_voxel_volume
            / 1000.0
        )

        soft01_volume_ml = (
            int(target_soft_01.sum())
            * output_voxel_volume
            / 1000.0
        )

        soft05_volume_ml = (
            int(target_soft_05.sum())
            * output_voxel_volume
            / 1000.0
        )

        soft_integral_volume_ml = (
            float(target_soft.sum())
            * output_voxel_volume
            / 1000.0
        )

        output_voxels = int(
            np.prod(
                reference_resampled.shape
            )
        )

        estimated_input_gb = (
            output_voxels
            * EXPECTED_MODEL_CHANNELS
            * 4
            / (1024 ** 3)
        )

        spacing_result_rows.append({
            'subject_id': subject_id,
            'dev_split': row['dev_split'],
            'lesion_volume_bin': (
                row['lesion_volume_bin']
            ),
            'original_orientation': (
                row['original_orientation']
            ),
            'candidate_name': candidate_name,
            'spacing_x': (
                candidate_spacing[0]
            ),
            'spacing_y': (
                candidate_spacing[1]
            ),
            'spacing_z': (
                candidate_spacing[2]
            ),
            'output_shape': tuple(
                int(value)
                for value
                in reference_resampled.shape
            ),
            'output_voxels': output_voxels,
            'estimated_12_channel_input_gb': (
                estimated_input_gb
            ),
            'source_target_volume_ml': (
                source_target_volume_ml
            ),
            'harmonized_target_volume_ml': (
                harmonized_target_volume_ml
            ),
            'native_target_nonempty': bool(
                target_harmonized.any()
            ),
            'native_target_voxels': (
                harmonized_target_voxels
            ),
            'nearest_target_voxels': int(
                target_nearest.sum()
            ),
            'nearest_target_volume_ml': (
                nearest_volume_ml
            ),
            'nearest_target_nonempty': bool(
                target_nearest.any()
            ),
            'nearest_volume_error_ml': (
                nearest_volume_ml
                - source_target_volume_ml
            ),
            'nearest_absolute_volume_error_ml': abs(
                nearest_volume_ml
                - source_target_volume_ml
            ),
            'nearest_back_dice': (
                binary_dice(
                    nearest_back,
                    target_harmonized,
                )
            ),
            'soft01_target_voxels': int(
                target_soft_01.sum()
            ),
            'soft01_target_volume_ml': (
                soft01_volume_ml
            ),
            'soft01_target_nonempty': bool(
                target_soft_01.any()
            ),
            'soft01_volume_error_ml': (
                soft01_volume_ml
                - source_target_volume_ml
            ),
            'soft01_absolute_volume_error_ml': abs(
                soft01_volume_ml
                - source_target_volume_ml
            ),
            'soft01_back_dice': (
                binary_dice(
                    soft01_back,
                    target_harmonized,
                )
            ),
            'soft05_target_voxels': int(
                target_soft_05.sum()
            ),
            'soft05_target_volume_ml': (
                soft05_volume_ml
            ),
            'soft05_target_nonempty': bool(
                target_soft_05.any()
            ),
            'soft05_volume_error_ml': (
                soft05_volume_ml
                - source_target_volume_ml
            ),
            'soft05_absolute_volume_error_ml': abs(
                soft05_volume_ml
                - source_target_volume_ml
            ),
            'soft05_back_dice': (
                binary_dice(
                    soft05_back,
                    target_harmonized,
                )
            ),
            'soft_integral_volume_ml': (
                soft_integral_volume_ml
            ),
            'soft_integral_absolute_volume_error_ml': abs(
                soft_integral_volume_ml
                - source_target_volume_ml
            ),
            'native_lvo_nonempty': bool(
                lvo_harmonized.any()
            ),
            'native_lvo_voxels': (
                harmonized_lvo_voxels
            ),
            'lvo_nearest_voxels': int(
                lvo_nearest.sum()
            ),
            'lvo_nearest_nonempty': bool(
                lvo_nearest.any()
            ),
            'lvo_soft01_voxels': int(
                lvo_soft_01.sum()
            ),
            'lvo_soft01_nonempty': bool(
                lvo_soft_01.any()
            ),
            'cow_native_voxels': (
                harmonized_cow_voxels
            ),
            'cow_soft01_voxels': int(
                cow_soft_01.sum()
            ),
            'cow_soft01_nonempty': bool(
                cow_soft_01.any()
            ),
            'target_coverage_by_resampled_support': (
                coverage_fraction(
                    target_soft_01,
                    support_resampled,
                )
            ),
        })

        if subject_id in qa_ids:
            candidate_snapshots[
                candidate_name
            ] = {
                'ncct': reference_ncct,
                'target_nearest': (
                    target_nearest.copy()
                ),
                'target_soft01': (
                    target_soft_01.copy()
                ),
                'support': (
                    support_resampled.copy()
                ),
                'cow': cow_soft_01.copy(),
                'lvo': lvo_soft_01.copy(),
            }

        del (
            reference_resampled,
            reference_ncct,
            target_soft,
            target_nearest,
            target_soft_01,
            target_soft_05,
            lvo_soft,
            lvo_nearest,
            lvo_soft_01,
            cow_soft,
            cow_soft_01,
            support_resampled,
            nearest_back,
            soft01_back,
            soft05_back,
        )

        gc.collect()

    
    
    

    if subject_id in qa_ids:
        row_count = (
            1 + len(CANDIDATE_SPACINGS)
        )

        figure, axes = plt.subplots(
            row_count,
            4,
            figsize=(
                18,
                4.5 * row_count,
            ),
            constrained_layout=True,
        )

        native_z = choose_display_slice(
            target_harmonized,
            fallback_mask=(
                cow_union_harmonized
            ),
        )

        axes[0, 0].imshow(
            display_plane(
                ncct,
                native_z,
            ),
            cmap='gray',
            vmin=0,
            vmax=100,
        )

        add_overlay(
            axes[0, 0],
            target_harmonized,
            native_z,
            cmap='Reds',
            alpha=0.55,
        )

        axes[0, 0].set_title(
            'Harmonized RAS: infarct'
        )
        axes[0, 0].axis('off')

        axes[0, 1].imshow(
            display_plane(
                tmax,
                native_z,
            ),
            cmap='magma',
            vmin=0,
            vmax=15,
        )

        add_overlay(
            axes[0, 1],
            target_harmonized,
            native_z,
            cmap='Greens',
            alpha=0.35,
        )

        axes[0, 1].set_title(
            'Harmonized Tmax + infarct'
        )
        axes[0, 1].axis('off')

        axes[0, 2].imshow(
            display_plane(
                ncct,
                native_z,
            ),
            cmap='gray',
            vmin=0,
            vmax=100,
        )

        add_overlay(
            axes[0, 2],
            support_harmonized,
            native_z,
            cmap='Greens',
            alpha=0.35,
        )

        axes[0, 2].set_title(
            'Provisional support mask'
        )
        axes[0, 2].axis('off')

        axes[0, 3].imshow(
            display_plane(
                ncct,
                native_z,
            ),
            cmap='gray',
            vmin=0,
            vmax=100,
        )

        add_overlay(
            axes[0, 3],
            cow_union_harmonized,
            native_z,
            cmap='winter',
            alpha=0.75,
        )

        add_overlay(
            axes[0, 3],
            lvo_harmonized,
            native_z,
            cmap='spring',
            alpha=0.90,
        )

        axes[0, 3].set_title(
            'Harmonized CoW + LVO'
        )
        axes[0, 3].axis('off')

        for plot_row, (
            candidate_name,
            snapshot,
        ) in enumerate(
            candidate_snapshots.items(),
            start=1,
        ):
            candidate_z = choose_display_slice(
                snapshot['target_soft01'],
                fallback_mask=snapshot['cow'],
            )

            axes[plot_row, 0].imshow(
                display_plane(
                    snapshot['ncct'],
                    candidate_z,
                ),
                cmap='gray',
                vmin=0,
                vmax=100,
            )

            add_overlay(
                axes[plot_row, 0],
                snapshot['target_nearest'],
                candidate_z,
                cmap='Reds',
                alpha=0.60,
            )

            axes[plot_row, 0].set_title(
                f'{candidate_name}: nearest target'
            )
            axes[plot_row, 0].axis('off')

            axes[plot_row, 1].imshow(
                display_plane(
                    snapshot['ncct'],
                    candidate_z,
                ),
                cmap='gray',
                vmin=0,
                vmax=100,
            )

            add_overlay(
                axes[plot_row, 1],
                snapshot['target_soft01'],
                candidate_z,
                cmap='Reds',
                alpha=0.60,
            )

            axes[plot_row, 1].set_title(
                f'{candidate_name}: soft target ≥ 0.10'
            )
            axes[plot_row, 1].axis('off')

            axes[plot_row, 2].imshow(
                display_plane(
                    snapshot['ncct'],
                    candidate_z,
                ),
                cmap='gray',
                vmin=0,
                vmax=100,
            )

            add_overlay(
                axes[plot_row, 2],
                snapshot['support'],
                candidate_z,
                cmap='Greens',
                alpha=0.35,
            )

            axes[plot_row, 2].set_title(
                f'{candidate_name}: support'
            )
            axes[plot_row, 2].axis('off')

            axes[plot_row, 3].imshow(
                display_plane(
                    snapshot['ncct'],
                    candidate_z,
                ),
                cmap='gray',
                vmin=0,
                vmax=100,
            )

            add_overlay(
                axes[plot_row, 3],
                snapshot['cow'],
                candidate_z,
                cmap='winter',
                alpha=0.75,
            )

            add_overlay(
                axes[plot_row, 3],
                snapshot['lvo'],
                candidate_z,
                cmap='spring',
                alpha=0.90,
            )

            axes[plot_row, 3].set_title(
                f'{candidate_name}: soft CoW + LVO'
            )
            axes[plot_row, 3].axis('off')

        figure.suptitle(
            f'{subject_id} | '
            f'{row["lesion_volume_bin"]} | '
            f'original={row["original_orientation"]} | '
            'canonical=RAS',
            fontsize=14,
        )

        qa_path = (
            QA_DIR
            / f'{subject_id}_spacing_comparison.png'
        )

        figure.savefig(
            qa_path,
            dpi=150,
            bbox_inches='tight',
        )

        plt.show()
        plt.close(figure)

        print(f'  Saved QA: {qa_path}')

    del (
        reference_image,
        source_images,
        arrays,
        ncct,
        cbf,
        cbv,
        mtt,
        tmax,
        target_harmonized,
        target_source,
        lvo_harmonized,
        lvo_source,
        cow_harmonized,
        cow_union_harmonized,
        cow_source,
        cow_union_source,
        support_harmonized,
        candidate_snapshots,
    )

    gc.collect()





harmonization_df = pd.DataFrame(
    harmonization_rows
)

native_df = pd.DataFrame(native_rows)

spacing_results_df = pd.DataFrame(
    spacing_result_rows
)

harmonization_path = (
    PHASE3_ROOT
    / 'source_grid_harmonization_audit.csv'
)

native_results_path = (
    PHASE3_ROOT
    / 'native_support_and_volume_audit.csv'
)

spacing_results_path = (
    PHASE3_ROOT
    / 'spacing_preservation_case_results.csv'
)

harmonization_df.to_csv(
    harmonization_path,
    index=False,
)

native_df.to_csv(
    native_results_path,
    index=False,
)

spacing_results_df.to_csv(
    spacing_results_path,
    index=False,
)

nonempty_target_df = (
    spacing_results_df[
        spacing_results_df[
            'native_target_nonempty'
        ]
    ].copy()
)

nonempty_lvo_df = (
    spacing_results_df[
        spacing_results_df[
            'native_lvo_nonempty'
        ]
    ].copy()
)

spacing_summary_df = (
    nonempty_target_df
    .groupby(
        [
            'candidate_name',
            'spacing_x',
            'spacing_y',
            'spacing_z',
        ],
        as_index=False,
    )
    .agg(
        nonempty_target_cases=(
            'subject_id',
            'nunique',
        ),
        nearest_target_retention=(
            'nearest_target_nonempty',
            'mean',
        ),
        soft01_target_retention=(
            'soft01_target_nonempty',
            'mean',
        ),
        soft05_target_retention=(
            'soft05_target_nonempty',
            'mean',
        ),
        median_nearest_back_dice=(
            'nearest_back_dice',
            'median',
        ),
        median_soft01_back_dice=(
            'soft01_back_dice',
            'median',
        ),
        median_soft05_back_dice=(
            'soft05_back_dice',
            'median',
        ),
        mean_nearest_absolute_volume_error_ml=(
            'nearest_absolute_volume_error_ml',
            'mean',
        ),
        mean_soft01_absolute_volume_error_ml=(
            'soft01_absolute_volume_error_ml',
            'mean',
        ),
        mean_soft05_absolute_volume_error_ml=(
            'soft05_absolute_volume_error_ml',
            'mean',
        ),
        mean_soft_integral_absolute_volume_error_ml=(
            'soft_integral_absolute_volume_error_ml',
            'mean',
        ),
        median_output_voxels=(
            'output_voxels',
            'median',
        ),
        median_estimated_12_channel_input_gb=(
            'estimated_12_channel_input_gb',
            'median',
        ),
        mean_target_coverage_by_support=(
            'target_coverage_by_resampled_support',
            'mean',
        ),
    )
)

if len(nonempty_lvo_df):
    lvo_summary_df = (
        nonempty_lvo_df
        .groupby(
            'candidate_name',
            as_index=False,
        )
        .agg(
            nonempty_lvo_cases=(
                'subject_id',
                'nunique',
            ),
            nearest_lvo_retention=(
                'lvo_nearest_nonempty',
                'mean',
            ),
            soft01_lvo_retention=(
                'lvo_soft01_nonempty',
                'mean',
            ),
        )
    )

    spacing_summary_df = (
        spacing_summary_df.merge(
            lvo_summary_df,
            on='candidate_name',
            how='left',
        )
    )

spacing_summary_path = (
    PHASE3_ROOT
    / 'spacing_preservation_summary.csv'
)

spacing_summary_df.to_csv(
    spacing_summary_path,
    index=False,
)





print('\nSource-grid harmonization summary:')

display(
    harmonization_df
    .groupby(
        [
            'role',
            'was_resampled_to_ncct',
            'large_displacement_warning',
        ],
        dropna=False,
    )
    .size()
    .reset_index(name='records')
)

print('\nSources requiring harmonization:')

display(
    harmonization_df[
        harmonization_df[
            'was_resampled_to_ncct'
        ]
    ][
        [
            'subject_id',
            'role',
            'source_shape',
            'reference_shape',
            'maximum_affine_element_difference',
            'maximum_corner_displacement_mm',
            'interpolation_order',
            'large_displacement_warning',
        ]
    ]
)

print('\nNative support-mask and harmonization summary:')

display(
    native_df[
        [
            'subject_id',
            'lesion_volume_bin',
            'original_orientation',
            'reference_spacing',
            'source_target_volume_ml',
            'harmonized_target_volume_ml',
            'harmonization_target_absolute_volume_error_ml',
            'target_coverage_by_support',
            'cow_coverage_by_support',
            'lvo_coverage_by_support',
            'estimated_midline_world_x_mm',
        ]
    ]
)

print('\nSpacing preservation summary:')
display(spacing_summary_df)

print('\nTiny-lesion results:')

display(
    spacing_results_df[
        spacing_results_df[
            'lesion_volume_bin'
        ] == 'tiny'
    ][
        [
            'subject_id',
            'candidate_name',
            'source_target_volume_ml',
            'native_target_voxels',
            'nearest_target_voxels',
            'nearest_target_nonempty',
            'soft01_target_voxels',
            'soft01_target_nonempty',
            'nearest_back_dice',
            'soft01_back_dice',
            'nearest_absolute_volume_error_ml',
            'soft01_absolute_volume_error_ml',
            'lvo_nearest_nonempty',
            'lvo_soft01_nonempty',
        ]
    ]
)

phase3_summary = {
    'candidate_spacings': {
        name: list(spacing)
        for name, spacing
        in CANDIDATE_SPACINGS.items()
    },
    'selected_subjects': selected_ids,
    'qa_subjects': qa_ids,
    'test_subjects_used': False,
    'canonical_reference': (
        'Each subject canonical RAS NCCT grid'
    ),
    'continuous_interpolation': (
        'trilinear, order=1'
    ),
    'mask_interpolation_for_grid_harmonization': (
        'nearest neighbor, order=0'
    ),
    'sources_harmonized_to_ncct': int(
        harmonization_df[
            'was_resampled_to_ncct'
        ].sum()
    ),
    'large_displacement_warnings': int(
        harmonization_df[
            'large_displacement_warning'
        ].sum()
    ),
    'final_spacing_selected': None,
    'final_target_strategy_selected': None,
    'note': (
        'This experiment does not create the final '
        'Trial8 dataset or train a model.'
    ),
}

phase3_summary_path = (
    PHASE3_ROOT
    / 'phase3_experiment_summary.json'
)

phase3_summary_path.write_text(
    json.dumps(
        phase3_summary,
        indent=2,
    ),
    encoding='utf-8',
)

print('\n' + '=' * 80)
print('REVISED TRIAL8 PHASE 3 EXPERIMENT COMPLETE')
print('=' * 80)
print(f'Spacing audit:          {spacing_path}')
print(f'Selected subjects:      {selected_path}')
print(f'Harmonization audit:    {harmonization_path}')
print(f'Native support audit:   {native_results_path}')
print(f'Case-level results:     {spacing_results_path}')
print(f'Spacing summary:        {spacing_summary_path}')
print(f'Visual QA directory:    {QA_DIR}')
print(f'Experiment summary:     {phase3_summary_path}')
print()
print('No final Trial8 dataset or model has been created.')


In [ ]:
!pip -q install nibabel scipy openpyxl

import gc
import json
import os
import random
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display
from nibabel.processing import resample_from_to, resample_to_output
from scipy import ndimage
from scipy.spatial import cKDTree





drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v1_trial0.csv'
)

PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2'
TARGET_SPACING = (0.8, 0.8, 2.0)
SEED = 20240824

PREPROCESSING_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
)

SUBJECT_OUTPUT_DIR = (
    PREPROCESSING_ROOT
    / 'subjects_npz'
)

CONFIG_DIR = PROJECT_ROOT / 'configs'
METADATA_DIR = PROJECT_ROOT / 'metadata'

AUDIT_PATH = (
    PREPROCESSING_ROOT
    / 'preprocessing_audit.csv'
)

PROGRESS_PATH = (
    PREPROCESSING_ROOT
    / 'preprocessing_progress.csv'
)

CONFIG_PATH = (
    CONFIG_DIR
    / f'{PREPROCESSING_VERSION}_config.json'
)

OUTPUT_MANIFEST_PATH = (
    METADATA_DIR
    / 'model_manifest_v2_vascufate.csv'
)

for output_directory in [
    PREPROCESSING_ROOT,
    SUBJECT_OUTPUT_DIR,
    CONFIG_DIR,
    METADATA_DIR,
]:
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

def assert_project_path(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()

    if (
        path != project_root
        and project_root not in path.parents
    ):
        raise ValueError(
            f'Unsafe path outside project root: {path}'
        )

for output_path in [
    PREPROCESSING_ROOT,
    SUBJECT_OUTPUT_DIR,
    CONFIG_PATH,
    AUDIT_PATH,
    PROGRESS_PATH,
    OUTPUT_MANIFEST_PATH,
]:
    assert_project_path(output_path)

random.seed(SEED)
np.random.seed(SEED)

if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        SOURCE_MANIFEST_PATH
    )

manifest_df = pd.read_csv(
    SOURCE_MANIFEST_PATH
)

SOURCE_COLUMNS = {
    'ncct': 'path_raw_ncct',
    'cta': 'path_deriv_cta',
    'cbf': 'path_deriv_cbf',
    'cbv': 'path_deriv_cbv',
    'mtt': 'path_deriv_mtt',
    'tmax': 'path_deriv_tmax',
    'cow': 'path_cow_mask',
    'lvo': 'path_lvo_mask',
    'target': 'path_final_infarct_mask',
}

required_columns = [
    'subject_id',
    'dev_split',
    'lesion_volume_bin',
    *SOURCE_COLUMNS.values(),
]

missing_columns = [
    column
    for column in required_columns
    if column not in manifest_df.columns
]

if missing_columns:
    raise ValueError(
        f'Missing required manifest columns: '
        f'{missing_columns}'
    )





INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

STRUCTURAL_CHANNEL_INDICES = [0, 1, 2]
PERFUSION_CHANNEL_INDICES = list(
    range(3, len(INPUT_CHANNELS))
)

VESSEL_NAMES = [
    'BA',
    'R_PCA',
    'L_PCA',
    'R_ICA',
    'R_MCA',
    'L_ICA',
    'L_MCA',
    'R_Pcom',
    'L_Pcom',
    'Acom',
    'R_ACA',
    'L_ACA',
    'third_A2',
]

VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

preprocessing_config = {
    'preprocessing_version': PREPROCESSING_VERSION,
    'source_manifest': str(
        SOURCE_MANIFEST_PATH
    ),
    'target_orientation': 'RAS',
    'target_spacing_mm': list(
        TARGET_SPACING
    ),
    'input_channels': INPUT_CHANNELS,
    'structural_channel_indices': (
        STRUCTURAL_CHANNEL_INDICES
    ),
    'perfusion_channel_indices': (
        PERFUSION_CHANNEL_INDICES
    ),
    'image_storage_dtype': 'float16',
    'target_binary_interpolation': (
        'nearest neighbor'
    ),
    'target_soft_interpolation': (
        'trilinear occupancy'
    ),
    'cow_label_interpolation': (
        'nearest neighbor'
    ),
    'cow_soft_union_interpolation': (
        'trilinear'
    ),
    'lvo_representation': (
        'native-resolution exponential '
        'distance proximity'
    ),
    'lvo_proximity_scale_mm': 10.0,
    'crop_uses_target': False,
    'crop_margin_mm': [
        16.0,
        16.0,
        10.0,
    ],
    'vessel_names': VESSEL_NAMES,
    'vessel_feature_names': (
        VESSEL_FEATURE_NAMES
    ),
    'seed': SEED,
    'flagged_microlesion': (
        'sub-stroke0171'
    ),
}

CONFIG_PATH.write_text(
    json.dumps(
        preprocessing_config,
        indent=2,
    ),
    encoding='utf-8',
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Manifest subjects: {len(manifest_df)}')
print(f'Preprocessing version: {PREPROCESSING_VERSION}')
print(f'Target spacing: {TARGET_SPACING}')
print(f'Image channels: {len(INPUT_CHANNELS)}')
print(f'Subject output directory: {SUBJECT_OUTPUT_DIR}')





def orientation_code(image):
    return ''.join(
        nib.aff2axcodes(image.affine)
    )

def canonical_image(path):
    image = nib.load(str(path))
    canonical = nib.as_closest_canonical(
        image
    )

    if orientation_code(canonical) != 'RAS':
        raise RuntimeError(
            f'Could not canonicalize to RAS: {path}'
        )

    return canonical

def image_array(image):
    return np.asarray(
        image.dataobj,
        dtype=np.float32,
    )

def voxel_volume_mm3(image):
    return float(
        abs(
            np.linalg.det(
                image.affine[:3, :3]
            )
        )
    )

def grids_match(first, second):
    return (
        tuple(first.shape)
        == tuple(second.shape)
        and np.allclose(
            first.affine,
            second.affine,
            atol=1e-4,
            rtol=1e-5,
        )
    )

def harmonize_image(
    image,
    reference,
    order,
    cval,
):
    if grids_match(image, reference):
        return image, False

    harmonized = resample_from_to(
        image,
        (
            reference.shape,
            reference.affine,
        ),
        order=order,
        mode='constant',
        cval=float(cval),
    )

    if not grids_match(
        harmonized,
        reference,
    ):
        raise RuntimeError(
            'Grid harmonization failed.'
        )

    return harmonized, True

def make_image(
    array,
    affine,
):
    return nib.Nifti1Image(
        np.asarray(
            array,
            dtype=np.float32,
        ),
        affine,
    )

def resample_array(
    array,
    source_affine,
    output_grid,
    order,
    cval=0.0,
):
    result = resample_from_to(
        make_image(
            array,
            source_affine,
        ),
        output_grid,
        order=order,
        mode='constant',
        cval=float(cval),
    )

    return np.asarray(
        result.dataobj,
        dtype=np.float32,
    )





def largest_component_overlapping(
    candidate,
    preferred_mask=None,
):
    candidate = np.asarray(
        candidate,
        dtype=bool,
    )

    labels, component_count = (
        ndimage.label(
            candidate,
            structure=(
                ndimage.generate_binary_structure(
                    3,
                    1,
                )
            ),
        )
    )

    if component_count == 0:
        return np.zeros_like(
            candidate,
            dtype=bool,
        )

    component_sizes = np.bincount(
        labels.ravel()
    )
    component_sizes[0] = 0

    selected_label = None

    if (
        preferred_mask is not None
        and np.asarray(
            preferred_mask,
            dtype=bool,
        ).any()
    ):
        preferred_mask = np.asarray(
            preferred_mask,
            dtype=bool,
        )

        overlap_counts = np.bincount(
            labels[preferred_mask].ravel(),
            minlength=component_count + 1,
        )
        overlap_counts[0] = 0

        if overlap_counts.max() > 0:
            selected_label = int(
                overlap_counts.argmax()
            )

    if selected_label is None:
        selected_label = int(
            component_sizes.argmax()
        )

    return labels == selected_label

def construct_perfusion_validity(
    ncct,
    cbf,
    cbv,
    mtt,
    tmax,
):
    finite = (
        np.isfinite(ncct)
        & np.isfinite(cbf)
        & np.isfinite(cbv)
        & np.isfinite(mtt)
        & np.isfinite(tmax)
    )

    ct_soft_tissue = (
        (ncct > -100.0)
        & (ncct < 200.0)
    )

    perfusion_defined = (
        (tmax > -29.5)
        | (cbf > 0.0)
        | (cbv > 0.0)
        | (mtt > 0.0)
    )

    validity = (
        finite
        & ct_soft_tissue
        & perfusion_defined
    )

    structure = (
        ndimage.generate_binary_structure(
            3,
            1,
        )
    )

    validity = ndimage.binary_closing(
        validity,
        structure=structure,
        iterations=2,
    )

    for z_index in range(
        validity.shape[2]
    ):
        validity[:, :, z_index] = (
            ndimage.binary_fill_holes(
                validity[:, :, z_index]
            )
        )

    validity = (
        largest_component_overlapping(
            validity
        )
    )

    validity = ndimage.binary_dilation(
        validity,
        structure=structure,
        iterations=1,
    )

    return validity.astype(bool)

def construct_broad_crop_mask(
    ncct,
    perfusion_valid,
    cow_union,
    lvo_proximity,
):
    finite = np.isfinite(ncct)

    head_candidate = (
        finite
        & (ncct > -500.0)
        & (ncct < 3000.0)
    )

    structure = (
        ndimage.generate_binary_structure(
            3,
            1,
        )
    )

    head_candidate = (
        ndimage.binary_closing(
            head_candidate,
            structure=structure,
            iterations=3,
        )
    )

    for z_index in range(
        head_candidate.shape[2]
    ):
        head_candidate[:, :, z_index] = (
            ndimage.binary_fill_holes(
                head_candidate[:, :, z_index]
            )
        )

    head_mask = (
        largest_component_overlapping(
            head_candidate,
            preferred_mask=perfusion_valid,
        )
    )

    head_mask = ndimage.binary_dilation(
        head_mask,
        structure=structure,
        iterations=2,
    )

    crop_mask = (
        head_mask
        | perfusion_valid
        | (cow_union > 0.01)
        | (lvo_proximity > 0.01)
    )

    crop_mask = ndimage.binary_closing(
        crop_mask,
        structure=structure,
        iterations=2,
    )

    return crop_mask.astype(bool)

def bounding_box_with_margin(
    mask,
    spacing,
    margin_mm=(16.0, 16.0, 10.0),
):
    coordinates = np.argwhere(mask)

    if len(coordinates) == 0:
        raise RuntimeError(
            'Crop mask is empty.'
        )

    minimum = coordinates.min(axis=0)
    maximum = coordinates.max(axis=0) + 1

    margin_voxels = np.ceil(
        np.asarray(margin_mm)
        / np.asarray(spacing)
    ).astype(int)

    minimum = np.maximum(
        minimum - margin_voxels,
        0,
    )

    maximum = np.minimum(
        maximum + margin_voxels,
        np.asarray(mask.shape),
    )

    return tuple(
        slice(
            int(minimum[axis]),
            int(maximum[axis]),
        )
        for axis in range(3)
    )

def cropped_affine(
    full_affine,
    crop_slices,
):
    crop_origin = np.asarray([
        crop_slices[0].start,
        crop_slices[1].start,
        crop_slices[2].start,
    ])

    output_affine = full_affine.copy()

    output_affine[:3, 3] = (
        nib.affines.apply_affine(
            full_affine,
            crop_origin,
        )
    )

    return output_affine

def mask_coverage(mask, container):
    mask = np.asarray(mask, dtype=bool)
    container = np.asarray(
        container,
        dtype=bool,
    )

    if not mask.any():
        return np.nan

    return float(
        np.logical_and(
            mask,
            container,
        ).sum()
        / mask.sum()
    )





def scaled_window(
    array,
    minimum,
    maximum,
    valid=None,
):
    output = np.clip(
        np.asarray(array, dtype=np.float32),
        minimum,
        maximum,
    )

    output = (
        output - minimum
    ) / (
        maximum - minimum
    )

    if valid is not None:
        output = np.where(
            valid,
            output,
            0.0,
        )

    return output.astype(np.float32)

def mirror_about_midline(
    array,
    midline_index,
):
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    x_indices = np.arange(
        array.shape[0]
    )

    mirrored_indices = np.rint(
        2.0 * midline_index
        - x_indices
    ).astype(int)

    index_valid = (
        (mirrored_indices >= 0)
        & (
            mirrored_indices
            < array.shape[0]
        )
    )

    output = np.zeros_like(
        array,
        dtype=np.float32,
    )

    output[index_valid] = (
        array[mirrored_indices[index_valid]]
    )

    return output, index_valid

def normalized_delta(
    value,
    mirror_value,
    scale,
    valid_pair,
):
    delta = (
        value - mirror_value
    ) / float(scale)

    delta = np.clip(
        delta,
        -1.0,
        1.0,
    )

    return np.where(
        valid_pair,
        delta,
        0.0,
    ).astype(np.float32)

def relative_ratio(
    value,
    mirror_value,
    denominator_floor,
    valid_pair,
):
    ratio = (
        value + denominator_floor
    ) / (
        mirror_value
        + denominator_floor
    )

    ratio = np.clip(
        ratio,
        0.0,
        2.0,
    ) / 2.0

    return np.where(
        valid_pair,
        ratio,
        0.0,
    ).astype(np.float32)





def world_coordinates(
    voxel_coordinates,
    affine,
):
    return nib.affines.apply_affine(
        affine,
        voxel_coordinates,
    )

def extract_vessel_features(
    cow_labels,
    lvo_mask,
    affine,
    voxel_volume,
    brain_center_world,
    cta,
    cbf,
    cbv,
    mtt,
    tmax,
):
    feature_matrix = np.zeros(
        (
            len(VESSEL_NAMES),
            len(VESSEL_FEATURE_NAMES),
        ),
        dtype=np.float32,
    )

    presence = np.zeros(
        len(VESSEL_NAMES),
        dtype=np.uint8,
    )

    lvo_coordinates = np.argwhere(
        lvo_mask
    )

    if len(lvo_coordinates):
        lvo_world = world_coordinates(
            lvo_coordinates,
            affine,
        )
        lvo_tree = cKDTree(lvo_world)
        lvo_present = True
    else:
        lvo_tree = None
        lvo_present = False

    zooms = np.sqrt(
        np.sum(
            affine[:3, :3] ** 2,
            axis=0,
        )
    )

    modality_arrays = [
        cta,
        cbf,
        cbv,
        mtt,
        tmax,
    ]

    for vessel_index, vessel_name in enumerate(
        VESSEL_NAMES,
        start=1,
    ):
        vessel_mask = (
            cow_labels == vessel_index
        )

        vessel_coordinates = np.argwhere(
            vessel_mask
        )

        if len(vessel_coordinates) == 0:
            continue

        matrix_index = vessel_index - 1
        presence[matrix_index] = 1

        centroid_voxel = (
            vessel_coordinates.mean(axis=0)
        )

        centroid_world = (
            world_coordinates(
                centroid_voxel,
                affine,
            )
        )

        minimum_voxel = (
            vessel_coordinates.min(axis=0)
        )
        maximum_voxel = (
            vessel_coordinates.max(axis=0)
        )

        extent_mm = (
            maximum_voxel
            - minimum_voxel
            + 1
        ) * zooms

        cta_values = cta[vessel_mask]

        perfusion_means = []

        for modality_array in modality_arrays[1:]:
            values = modality_array[
                vessel_mask
            ]
            values = values[
                np.isfinite(values)
            ]

            if len(values):
                perfusion_means.append(
                    float(values.mean())
                )
            else:
                perfusion_means.append(0.0)

        if lvo_present:
            vessel_world = world_coordinates(
                vessel_coordinates,
                affine,
            )

            minimum_lvo_distance = float(
                lvo_tree.query(
                    vessel_world,
                    k=1,
                )[0].min()
            )
        else:
            minimum_lvo_distance = 100.0

        overlap_fraction = float(
            np.logical_and(
                vessel_mask,
                lvo_mask,
            ).sum()
            / vessel_mask.sum()
        )

        feature_matrix[matrix_index] = [
            1.0,
            float(
                vessel_mask.sum()
                * voxel_volume
                / 1000.0
            ),
            float(
                centroid_world[0]
                - brain_center_world[0]
            ),
            float(
                centroid_world[1]
                - brain_center_world[1]
            ),
            float(
                centroid_world[2]
                - brain_center_world[2]
            ),
            float(extent_mm[0]),
            float(extent_mm[1]),
            float(extent_mm[2]),
            float(cta_values.mean()),
            float(
                np.percentile(
                    cta_values,
                    90,
                )
            ),
            perfusion_means[0],
            perfusion_means[1],
            perfusion_means[2],
            perfusion_means[3],
            minimum_lvo_distance,
            overlap_fraction,
        ]

    return (
        feature_matrix,
        presence,
        int(lvo_present),
    )





REQUIRED_NPZ_KEYS = {
    'image',
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'crop_mask',
    'perfusion_valid',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'full_resampled_affine',
    'crop_origin',
    'full_resampled_shape',
    'subject_id',
    'preprocessing_version',
}

def validate_output_npz(path):
    result = {
        'valid': False,
        'error': None,
    }

    try:
        with np.load(
            path,
            allow_pickle=False,
        ) as data:
            missing_keys = (
                REQUIRED_NPZ_KEYS
                - set(data.files)
            )

            if missing_keys:
                raise ValueError(
                    f'Missing NPZ keys: '
                    f'{sorted(missing_keys)}'
                )

            image = data['image']
            target_binary = (
                data['target_binary']
            )
            target_soft = (
                data['target_soft']
            )
            cow_label = data['cow_label']
            lvo_proximity = (
                data['lvo_proximity']
            )

            spatial_shape = tuple(
                image.shape[1:]
            )

            if image.shape[0] != len(
                INPUT_CHANNELS
            ):
                raise ValueError(
                    f'Expected '
                    f'{len(INPUT_CHANNELS)} channels; '
                    f'found {image.shape[0]}.'
                )

            for name, array in [
                (
                    'target_binary',
                    target_binary,
                ),
                ('target_soft', target_soft),
                ('cow_label', cow_label),
                (
                    'lvo_proximity',
                    lvo_proximity,
                ),
            ]:
                if tuple(array.shape) != spatial_shape:
                    raise ValueError(
                        f'{name} shape mismatch: '
                        f'{array.shape} vs '
                        f'{spatial_shape}'
                    )

            if not np.isfinite(image).all():
                raise ValueError(
                    'Image contains nonfinite values.'
                )

            if not np.isfinite(
                target_soft
            ).all():
                raise ValueError(
                    'Soft target contains '
                    'nonfinite values.'
                )

            if not np.isfinite(
                lvo_proximity
            ).all():
                raise ValueError(
                    'LVO proximity contains '
                    'nonfinite values.'
                )

            if image.min() < -1.0001:
                raise ValueError(
                    'Image minimum is below -1.'
                )

            if image.max() > 1.0001:
                raise ValueError(
                    'Image maximum is above 1.'
                )

            if (
                target_binary.min() < 0
                or target_binary.max() > 1
            ):
                raise ValueError(
                    'Binary target is invalid.'
                )

            result.update({
                'valid': True,
                'image_shape': tuple(
                    int(value)
                    for value in image.shape
                ),
                'target_binary_voxels': int(
                    target_binary.sum()
                ),
                'target_soft_sum': float(
                    target_soft.sum()
                ),
                'lvo_present': int(
                    data['lvo_present']
                ),
                'vessels_present': int(
                    data[
                        'vessel_presence'
                    ].sum()
                ),
                'binary_loss_weight': float(
                    data[
                        'binary_loss_weight'
                    ]
                ),
            })

    except Exception as exc:
        result['error'] = repr(exc)

    return result





def process_subject(row, output_path):
    subject_id = row['subject_id']

    source_images = {
        role: canonical_image(
            row[column]
        )
        for role, column
        in SOURCE_COLUMNS.items()
    }

    reference_native = (
        source_images['ncct']
    )

    harmonized_images = {
        'ncct': reference_native
    }

    harmonized_roles = []

    for role in [
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'cow',
        'lvo',
        'target',
    ]:
        order = (
            1
            if role in {
                'cta',
                'cbf',
                'cbv',
                'mtt',
                'tmax',
            }
            else 0
        )

        cval = {
            'cta': -1000.0,
            'cbf': 0.0,
            'cbv': 0.0,
            'mtt': 0.0,
            'tmax': -30.0,
            'cow': 0.0,
            'lvo': 0.0,
            'target': 0.0,
        }[role]

        harmonized, was_resampled = (
            harmonize_image(
                source_images[role],
                reference_native,
                order=order,
                cval=cval,
            )
        )

        harmonized_images[role] = (
            harmonized
        )

        if was_resampled:
            harmonized_roles.append(role)

    native_arrays = {
        role: image_array(image)
        for role, image
        in harmonized_images.items()
    }

    ncct_native = native_arrays['ncct']
    cta_native = native_arrays['cta']
    cbf_native = native_arrays['cbf']
    cbv_native = native_arrays['cbv']
    mtt_native = native_arrays['mtt']
    tmax_native = native_arrays['tmax']

    cow_native = np.rint(
        native_arrays['cow']
    ).astype(np.int16)

    lvo_native = (
        native_arrays['lvo'] > 0.5
    )

    target_native = (
        native_arrays['target'] > 0.5
    )

    target_source_array = (
        image_array(
            source_images['target']
        ) > 0
    )

    source_target_volume_ml = float(
        target_source_array.sum()
        * voxel_volume_mm3(
            source_images['target']
        )
        / 1000.0
    )

    native_perfusion_valid = (
        construct_perfusion_validity(
            ncct=ncct_native,
            cbf=cbf_native,
            cbv=cbv_native,
            mtt=mtt_native,
            tmax=tmax_native,
        )
    )

    native_valid_coordinates = (
        np.argwhere(
            native_perfusion_valid
        )
    )

    if len(native_valid_coordinates):
        native_center_voxel = (
            native_valid_coordinates.min(axis=0)
            + native_valid_coordinates.max(
                axis=0
            )
        ) / 2.0
    else:
        native_center_voxel = (
            np.asarray(
                reference_native.shape
            ) - 1
        ) / 2.0

    brain_center_world = (
        nib.affines.apply_affine(
            reference_native.affine,
            native_center_voxel,
        )
    )

    (
        vessel_features,
        vessel_presence,
        lvo_present,
    ) = extract_vessel_features(
        cow_labels=cow_native,
        lvo_mask=lvo_native,
        affine=reference_native.affine,
        voxel_volume=voxel_volume_mm3(
            reference_native
        ),
        brain_center_world=(
            brain_center_world
        ),
        cta=cta_native,
        cbf=cbf_native,
        cbv=cbv_native,
        mtt=mtt_native,
        tmax=tmax_native,
    )

    if lvo_present:
        native_spacing = (
            reference_native.header
            .get_zooms()[:3]
        )

        native_lvo_distance = (
            ndimage.distance_transform_edt(
                ~lvo_native,
                sampling=native_spacing,
            )
        ).astype(np.float32)

        native_lvo_proximity = np.exp(
            -native_lvo_distance / 10.0
        ).astype(np.float32)
    else:
        native_lvo_proximity = (
            np.zeros_like(
                ncct_native,
                dtype=np.float32,
            )
        )

    output_reference = (
        resample_to_output(
            reference_native,
            voxel_sizes=TARGET_SPACING,
            order=1,
            mode='constant',
            cval=-1000.0,
        )
    )

    output_grid = (
        output_reference.shape,
        output_reference.affine,
    )

    ncct = image_array(
        output_reference
    )

    cta = resample_array(
        cta_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=-1000.0,
    )

    cbf = resample_array(
        cbf_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=0.0,
    )

    cbv = resample_array(
        cbv_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=0.0,
    )

    mtt = resample_array(
        mtt_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=0.0,
    )

    tmax = resample_array(
        tmax_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=-30.0,
    )

    target_binary = (
        resample_array(
            target_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        ) > 0.5
    )

    target_soft = np.clip(
        resample_array(
            target_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    cow_label = np.rint(
        resample_array(
            cow_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        )
    ).astype(np.uint8)

    cow_soft_union = np.clip(
        resample_array(
            (cow_native > 0).astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    lvo_proximity = np.clip(
        resample_array(
            native_lvo_proximity,
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    perfusion_valid = (
        resample_array(
            native_perfusion_valid.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        ) > 0.5
    )

    crop_mask_full = (
        construct_broad_crop_mask(
            ncct=ncct,
            perfusion_valid=(
                perfusion_valid
            ),
            cow_union=cow_soft_union,
            lvo_proximity=lvo_proximity,
        )
    )

    crop_slices = (
        bounding_box_with_margin(
            crop_mask_full,
            spacing=TARGET_SPACING,
            margin_mm=(
                16.0,
                16.0,
                10.0,
            ),
        )
    )

    crop_origin = np.asarray([
        crop_slices[0].start,
        crop_slices[1].start,
        crop_slices[2].start,
    ], dtype=np.int32)

    crop_mask = (
        crop_mask_full[crop_slices]
    )

    perfusion_valid_crop = (
        perfusion_valid[crop_slices]
    )

    target_binary_crop = (
        target_binary[crop_slices]
    )

    target_soft_crop = (
        target_soft[crop_slices]
    )

    cow_label_crop = (
        cow_label[crop_slices]
    )

    cow_soft_union_crop = (
        cow_soft_union[crop_slices]
    )

    lvo_proximity_crop = (
        lvo_proximity[crop_slices]
    )

    
    crop_coordinates = np.argwhere(
        crop_mask_full
    )

    midline_index = float(
        (
            crop_coordinates[:, 0].min()
            + crop_coordinates[:, 0].max()
        ) / 2.0
    )

    mirror_cbf, mirror_index_valid = (
        mirror_about_midline(
            cbf,
            midline_index,
        )
    )

    mirror_cbv, _ = mirror_about_midline(
        cbv,
        midline_index,
    )

    mirror_mtt, _ = mirror_about_midline(
        mtt,
        midline_index,
    )

    mirror_tmax, _ = mirror_about_midline(
        tmax,
        midline_index,
    )

    mirror_valid_float, _ = (
        mirror_about_midline(
            perfusion_valid.astype(
                np.float32
            ),
            midline_index,
        )
    )

    mirror_valid = (
        mirror_valid_float > 0.5
    )

    index_valid_volume = np.broadcast_to(
        mirror_index_valid[
            :,
            None,
            None,
        ],
        perfusion_valid.shape,
    )

    pair_valid = (
        perfusion_valid
        & mirror_valid
        & index_valid_volume
    )

    relative_cbf = relative_ratio(
        cbf,
        mirror_cbf,
        denominator_floor=5.0,
        valid_pair=pair_valid,
    )

    relative_cbv = relative_ratio(
        cbv,
        mirror_cbv,
        denominator_floor=0.5,
        valid_pair=pair_valid,
    )

    delta_cbf = normalized_delta(
        cbf,
        mirror_cbf,
        scale=100.0,
        valid_pair=pair_valid,
    )

    delta_cbv = normalized_delta(
        cbv,
        mirror_cbv,
        scale=10.0,
        valid_pair=pair_valid,
    )

    delta_mtt = normalized_delta(
        mtt,
        mirror_mtt,
        scale=20.0,
        valid_pair=pair_valid,
    )

    delta_tmax = normalized_delta(
        tmax,
        mirror_tmax,
        scale=15.0,
        valid_pair=pair_valid,
    )

    channel_arrays = [
        scaled_window(
            ncct,
            -20.0,
            100.0,
        )[crop_slices],
        scaled_window(
            ncct,
            -100.0,
            300.0,
        )[crop_slices],
        scaled_window(
            cta,
            -100.0,
            1000.0,
        )[crop_slices],
        scaled_window(
            cbf,
            0.0,
            150.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            cbv,
            0.0,
            10.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            mtt,
            0.0,
            30.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            tmax,
            0.0,
            20.0,
            valid=perfusion_valid,
        )[crop_slices],
        relative_cbf[crop_slices],
        relative_cbv[crop_slices],
        delta_cbf[crop_slices],
        delta_cbv[crop_slices],
        delta_mtt[crop_slices],
        delta_tmax[crop_slices],
        perfusion_valid_crop.astype(
            np.float32
        ),
        pair_valid[crop_slices].astype(
            np.float32
        ),
    ]

    image = np.stack(
        channel_arrays,
        axis=0,
    ).astype(np.float16)

    output_affine = cropped_affine(
        output_reference.affine,
        crop_slices,
    )

    output_voxel_volume = (
        voxel_volume_mm3(
            output_reference
        )
    )

    target_binary_volume_ml = float(
        target_binary_crop.sum()
        * output_voxel_volume
        / 1000.0
    )

    target_soft_volume_ml = float(
        target_soft_crop.sum()
        * output_voxel_volume
        / 1000.0
    )

    source_target_nonempty = (
        source_target_volume_ml > 0
    )

    binary_target_nonempty = bool(
        target_binary_crop.any()
    )

    binary_loss_weight = 1.0

    if (
        source_target_nonempty
        and not binary_target_nonempty
    ):
        binary_loss_weight = 0.0

    target_crop_coverage = (
        mask_coverage(
            target_binary,
            crop_mask_full,
        )
    )

    full_resampled_shape = np.asarray(
        output_reference.shape,
        dtype=np.int32,
    )

    temporary_path = Path(
        str(output_path) + '.tmp'
    )

    with temporary_path.open('wb') as file:
        np.savez_compressed(
            file,
            image=image,
            target_binary=(
                target_binary_crop.astype(
                    np.uint8
                )
            ),
            target_soft=(
                target_soft_crop.astype(
                    np.float16
                )
            ),
            binary_loss_weight=np.asarray(
                binary_loss_weight,
                dtype=np.float32,
            ),
            crop_mask=(
                crop_mask.astype(
                    np.uint8
                )
            ),
            perfusion_valid=(
                perfusion_valid_crop.astype(
                    np.uint8
                )
            ),
            cow_label=(
                cow_label_crop.astype(
                    np.uint8
                )
            ),
            cow_soft_union=(
                cow_soft_union_crop.astype(
                    np.float16
                )
            ),
            lvo_proximity=(
                lvo_proximity_crop.astype(
                    np.float16
                )
            ),
            lvo_present=np.asarray(
                lvo_present,
                dtype=np.uint8,
            ),
            vessel_features_raw=(
                vessel_features.astype(
                    np.float32
                )
            ),
            vessel_presence=(
                vessel_presence.astype(
                    np.uint8
                )
            ),
            input_channels=np.asarray(
                INPUT_CHANNELS
            ),
            structural_channel_indices=(
                np.asarray(
                    STRUCTURAL_CHANNEL_INDICES,
                    dtype=np.int16,
                )
            ),
            perfusion_channel_indices=(
                np.asarray(
                    PERFUSION_CHANNEL_INDICES,
                    dtype=np.int16,
                )
            ),
            vessel_names=np.asarray(
                VESSEL_NAMES
            ),
            vessel_feature_names=np.asarray(
                VESSEL_FEATURE_NAMES
            ),
            affine=output_affine.astype(
                np.float64
            ),
            full_resampled_affine=(
                output_reference.affine.astype(
                    np.float64
                )
            ),
            crop_origin=crop_origin,
            full_resampled_shape=(
                full_resampled_shape
            ),
            target_spacing=np.asarray(
                TARGET_SPACING,
                dtype=np.float32,
            ),
            subject_id=np.asarray(
                subject_id
            ),
            preprocessing_version=np.asarray(
                PREPROCESSING_VERSION
            ),
            source_target_volume_ml=np.asarray(
                source_target_volume_ml,
                dtype=np.float32,
            ),
            target_binary_volume_ml=np.asarray(
                target_binary_volume_ml,
                dtype=np.float32,
            ),
            target_soft_volume_ml=np.asarray(
                target_soft_volume_ml,
                dtype=np.float32,
            ),
            target_crop_coverage=np.asarray(
                target_crop_coverage,
                dtype=np.float32,
            ),
            estimated_midline_full_index=np.asarray(
                midline_index,
                dtype=np.float32,
            ),
            harmonized_roles=np.asarray(
                harmonized_roles
            ),
        )

    os.replace(
        temporary_path,
        output_path,
    )

    result = validate_output_npz(
        output_path
    )

    if not result['valid']:
        raise RuntimeError(
            f'Output validation failed: '
            f'{result["error"]}'
        )

    result.update({
        'subject_id': subject_id,
        'output_path': str(output_path),
        'source_target_volume_ml': (
            source_target_volume_ml
        ),
        'target_binary_volume_ml': (
            target_binary_volume_ml
        ),
        'target_soft_volume_ml': (
            target_soft_volume_ml
        ),
        'target_crop_coverage': (
            target_crop_coverage
        ),
        'crop_shape': tuple(
            int(value)
            for value in image.shape[1:]
        ),
        'harmonized_roles': (
            harmonized_roles
        ),
        'status': 'processed',
    })

    del (
        source_images,
        harmonized_images,
        native_arrays,
        ncct_native,
        cta_native,
        cbf_native,
        cbv_native,
        mtt_native,
        tmax_native,
        cow_native,
        lvo_native,
        target_native,
        target_source_array,
        native_perfusion_valid,
        native_lvo_proximity,
        output_reference,
        ncct,
        cta,
        cbf,
        cbv,
        mtt,
        tmax,
        target_binary,
        target_soft,
        cow_label,
        cow_soft_union,
        lvo_proximity,
        perfusion_valid,
        crop_mask_full,
        mirror_cbf,
        mirror_cbv,
        mirror_mtt,
        mirror_tmax,
        mirror_valid_float,
        mirror_valid,
        pair_valid,
        relative_cbf,
        relative_cbv,
        delta_cbf,
        delta_cbv,
        delta_mtt,
        delta_tmax,
        channel_arrays,
        image,
    )

    gc.collect()

    return result





audit_rows = []

for subject_index, (_, row) in enumerate(
    manifest_df
    .sort_values('subject_id')
    .iterrows(),
    start=1,
):
    subject_id = row['subject_id']

    output_path = (
        SUBJECT_OUTPUT_DIR
        / (
            f'{subject_id}_'
            f'{PREPROCESSING_VERSION}.npz'
        )
    )

    print(
        f'\n{subject_index}/{len(manifest_df)} '
        f'{subject_id}'
    )

    audit_record = {
        'subject_id': subject_id,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': (
            row['lesion_volume_bin']
        ),
        'output_path': str(output_path),
        'status': None,
        'valid': False,
        'error': None,
    }

    try:
        if output_path.exists():
            existing_result = (
                validate_output_npz(
                    output_path
                )
            )

            if existing_result['valid']:
                audit_record.update(
                    existing_result
                )
                audit_record['status'] = (
                    'skipped_valid_existing'
                )

                print(
                    '  Valid existing output found. '
                    'Skipping.'
                )
            else:
                print(
                    '  Existing output is invalid. '
                    'Regenerating.'
                )

                processed_result = (
                    process_subject(
                        row,
                        output_path,
                    )
                )

                audit_record.update(
                    processed_result
                )
        else:
            processed_result = process_subject(
                row,
                output_path,
            )

            audit_record.update(
                processed_result
            )

        print(
            f'  Status: '
            f'{audit_record["status"]}'
        )
        print(
            f'  Image shape: '
            f'{audit_record.get("image_shape")}'
        )
        print(
            f'  Binary target voxels: '
            f'{audit_record.get("target_binary_voxels")}'
        )
        print(
            f'  Vessels present: '
            f'{audit_record.get("vessels_present")}'
        )

    except Exception as exc:
        audit_record.update({
            'status': 'failed',
            'valid': False,
            'error': repr(exc),
        })

        print(f'  FAILED: {exc}')

    audit_rows.append(
        audit_record
    )

    pd.DataFrame(
        audit_rows
    ).to_csv(
        PROGRESS_PATH,
        index=False,
    )

    gc.collect()





final_audit_rows = []

for _, row in (
    manifest_df
    .sort_values('subject_id')
    .iterrows()
):
    subject_id = row['subject_id']

    output_path = (
        SUBJECT_OUTPUT_DIR
        / (
            f'{subject_id}_'
            f'{PREPROCESSING_VERSION}.npz'
        )
    )

    validation = validate_output_npz(
        output_path
    ) if output_path.exists() else {
        'valid': False,
        'error': 'Output file missing.',
    }

    final_audit_rows.append({
        'subject_id': subject_id,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': (
            row['lesion_volume_bin']
        ),
        'output_path': str(output_path),
        **validation,
    })

audit_df = pd.DataFrame(
    final_audit_rows
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False,
)

valid_audit_df = audit_df[
    audit_df['valid']
].copy()

output_manifest_df = (
    manifest_df.merge(
        valid_audit_df[
            [
                'subject_id',
                'output_path',
                'image_shape',
                'target_binary_voxels',
                'target_soft_sum',
                'lvo_present',
                'vessels_present',
                'binary_loss_weight',
            ]
        ],
        on='subject_id',
        how='inner',
        validate='one_to_one',
    )
)

output_manifest_df = (
    output_manifest_df.rename(
        columns={
            'output_path': (
                'vascufate_v2_npz_path'
            ),
            'image_shape': (
                'vascufate_v2_image_shape'
            ),
            'target_binary_voxels': (
                'vascufate_v2_target_binary_voxels'
            ),
            'target_soft_sum': (
                'vascufate_v2_target_soft_sum'
            ),
            'lvo_present': (
                'vascufate_v2_lvo_present'
            ),
            'vessels_present': (
                'vascufate_v2_vessels_present'
            ),
            'binary_loss_weight': (
                'vascufate_v2_binary_loss_weight'
            ),
        }
    )
)

output_manifest_df.to_csv(
    OUTPUT_MANIFEST_PATH,
    index=False,
)

print('\n' + '=' * 80)
print('TRIAL8 V2 CORE PREPROCESSING COMPLETE')
print('=' * 80)

print(f'Total expected subjects: {len(manifest_df)}')
print(f'Valid outputs: {len(valid_audit_df)}')
print(
    f'Failed or missing outputs: '
    f'{len(manifest_df) - len(valid_audit_df)}'
)

print('\nStatus counts:')
display(
    audit_df[
        'valid'
    ].value_counts(
        dropna=False
    ).rename_axis(
        'valid'
    ).reset_index(
        name='subjects'
    )
)

print('\nSplit counts in v2 manifest:')
display(
    output_manifest_df
    .groupby('dev_split')
    .size()
    .reset_index(name='subjects')
)

print('\nBinary loss-weight counts:')
display(
    output_manifest_df[
        'vascufate_v2_binary_loss_weight'
    ].value_counts(
        dropna=False
    ).rename_axis(
        'binary_loss_weight'
    ).reset_index(
        name='subjects'
    )
)

print('\nSubjects with empty binary targets:')
display(
    output_manifest_df[
        output_manifest_df[
            'vascufate_v2_target_binary_voxels'
        ] == 0
    ][
        [
            'subject_id',
            'dev_split',
            'lesion_volume_bin',
            'vascufate_v2_target_soft_sum',
            'vascufate_v2_binary_loss_weight',
        ]
    ]
)

print('\nInvalid outputs:')
display(
    audit_df[
        ~audit_df['valid']
    ][
        [
            'subject_id',
            'output_path',
            'error',
        ]
    ]
)

print(f'\nSaved configuration: {CONFIG_PATH}')
print(f'Saved audit: {AUDIT_PATH}')
print(f'Saved v2 manifest: {OUTPUT_MANIFEST_PATH}')
print(f'Subject outputs: {SUBJECT_OUTPUT_DIR}')

if len(valid_audit_df) != len(
    manifest_df
):
    print(
        '\nSome subjects failed. Rerun this same '
        'standalone cell to resume only missing or '
        'invalid subjects.'
    )
else:
    print(
        '\nAll subjects completed and passed '
        'NPZ validation.'
    )

print(
    '\nNo Trial8 model has been trained. '
    'The vascular-territory atlas sidecars '
    'are the next stage.'
)


In [ ]:
!pip -q install nibabel

import json
import warnings
import numpy as np
import pandas as pd
import nibabel as nib

from pathlib import Path
from google.colab import drive

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v1_trial0.csv'
)
V2_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / 'v2_vascufate_ras_0p8x0p8x2'
)
V2_SUBJECT_DIR = V2_ROOT / 'subjects_npz'
DIAGNOSTIC_DIR = V2_ROOT / 'numeric_failure_diagnostic'

DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CHANNEL_NAMES = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

PERFUSION_COLUMNS = {
    'cbf': 'path_deriv_cbf',
    'cbv': 'path_deriv_cbv',
    'mtt': 'path_deriv_mtt',
    'tmax': 'path_deriv_tmax',
}

MAX_PERCENTILE_SAMPLE = 1_000_000
EXTREME_ABSOLUTE_THRESHOLD = 1e6
RNG = np.random.default_rng(20240824)


def sampled_percentiles(values, percentiles):
    values = np.asarray(values)

    if values.size == 0:
        return [np.nan] * len(percentiles)

    if values.size > MAX_PERCENTILE_SAMPLE:
        indices = RNG.choice(
            values.size,
            size=MAX_PERCENTILE_SAMPLE,
            replace=False,
        )
        values = values[indices]

    values = values.astype(np.float64, copy=False)
    return np.percentile(values, percentiles).tolist()


def scalar_text(value):
    value = np.asarray(value)

    if value.shape == ():
        return str(value.item())

    return str(value.reshape(-1)[0])


def find_key(keys, candidates):
    key_lookup = {str(key).lower(): str(key) for key in keys}

    for candidate in candidates:
        if candidate.lower() in key_lookup:
            return key_lookup[candidate.lower()]

    return None


if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Missing source manifest: {SOURCE_MANIFEST_PATH}'
    )

source_df = pd.read_csv(SOURCE_MANIFEST_PATH)

if len(source_df) != 148:
    raise RuntimeError(
        f'Expected 148 manifest subjects, found {len(source_df)}.'
    )

print(f'Manifest subjects: {len(source_df)}')
print(f'Existing v2 NPZ files: {len(list(V2_SUBJECT_DIR.glob("*.npz")))}')
print(f'Diagnostic output: {DIAGNOSTIC_DIR}')






raw_rows = []

for subject_index, row in source_df.iterrows():
    subject_id = row['subject_id']

    for modality, column in PERFUSION_COLUMNS.items():
        source_path = Path(str(row[column]))

        result = {
            'subject_id': subject_id,
            'modality': modality,
            'source_column': column,
            'path': str(source_path),
            'readable': False,
            'shape': None,
            'dtype': None,
            'total_voxels': None,
            'finite_voxels': None,
            'nan_voxels': None,
            'positive_inf_voxels': None,
            'negative_inf_voxels': None,
            'nonfinite_voxels': None,
            'extreme_abs_gt_1e6': None,
            'finite_min': None,
            'finite_p001': None,
            'finite_p01': None,
            'finite_p50': None,
            'finite_p99': None,
            'finite_p999': None,
            'finite_max': None,
            'finite_mean_float64': None,
            'error': None,
        }

        try:
            image = nib.load(str(source_path))
            data = np.asanyarray(image.dataobj).astype(
                np.float32,
                copy=False,
            )

            finite_mask = np.isfinite(data)
            finite_values = data[finite_mask]

            nan_count = int(np.isnan(data).sum())
            posinf_count = int(np.isposinf(data).sum())
            neginf_count = int(np.isneginf(data).sum())
            nonfinite_count = int((~finite_mask).sum())

            if finite_values.size:
                percentiles = sampled_percentiles(
                    finite_values,
                    [0.1, 1, 50, 99, 99.9],
                )

                finite_min = float(
                    np.min(finite_values).astype(np.float64)
                )
                finite_max = float(
                    np.max(finite_values).astype(np.float64)
                )
                finite_mean = float(
                    np.mean(finite_values, dtype=np.float64)
                )
                extreme_count = int(
                    np.count_nonzero(
                        np.abs(
                            finite_values.astype(
                                np.float64,
                                copy=False,
                            )
                        ) > EXTREME_ABSOLUTE_THRESHOLD
                    )
                )
            else:
                percentiles = [np.nan] * 5
                finite_min = np.nan
                finite_max = np.nan
                finite_mean = np.nan
                extreme_count = 0

            result.update({
                'readable': True,
                'shape': str(tuple(data.shape)),
                'dtype': str(data.dtype),
                'total_voxels': int(data.size),
                'finite_voxels': int(finite_mask.sum()),
                'nan_voxels': nan_count,
                'positive_inf_voxels': posinf_count,
                'negative_inf_voxels': neginf_count,
                'nonfinite_voxels': nonfinite_count,
                'extreme_abs_gt_1e6': extreme_count,
                'finite_min': finite_min,
                'finite_p001': percentiles[0],
                'finite_p01': percentiles[1],
                'finite_p50': percentiles[2],
                'finite_p99': percentiles[3],
                'finite_p999': percentiles[4],
                'finite_max': finite_max,
                'finite_mean_float64': finite_mean,
            })

        except Exception as exc:
            result['error'] = repr(exc)

        raw_rows.append(result)

    if (
        subject_index == 0
        or (subject_index + 1) % 20 == 0
        or subject_index + 1 == len(source_df)
    ):
        print(
            f'Audited raw perfusion inputs: '
            f'{subject_index + 1}/{len(source_df)} subjects'
        )

raw_audit_df = pd.DataFrame(raw_rows)

raw_audit_path = (
    DIAGNOSTIC_DIR / 'raw_perfusion_numeric_audit.csv'
)
raw_audit_df.to_csv(raw_audit_path, index=False)

raw_problem_df = raw_audit_df[
    (~raw_audit_df['readable'])
    | (raw_audit_df['nonfinite_voxels'].fillna(0) > 0)
    | (raw_audit_df['extreme_abs_gt_1e6'].fillna(0) > 0)
].copy()

print('\nRaw inputs with nonfinite or extreme values:')
if len(raw_problem_df):
    display(
        raw_problem_df[
            [
                'subject_id',
                'modality',
                'nan_voxels',
                'positive_inf_voxels',
                'negative_inf_voxels',
                'extreme_abs_gt_1e6',
                'finite_min',
                'finite_p001',
                'finite_p999',
                'finite_max',
                'error',
            ]
        ].sort_values(['subject_id', 'modality'])
    )
else:
    print('None detected.')






npz_paths = sorted(V2_SUBJECT_DIR.glob('*.npz'))

npz_case_rows = []
npz_channel_rows = []
schema_rows = []

for file_index, npz_path in enumerate(npz_paths, start=1):
    case_result = {
        'path': str(npz_path),
        'subject_id': npz_path.name.split('_preprocessed')[0],
        'readable': False,
        'image_key': None,
        'image_shape': None,
        'image_dtype': None,
        'image_channels': None,
        'image_nonfinite_voxels': None,
        'maximum_absolute_finite_value': None,
        'error': None,
    }

    try:
        with np.load(npz_path, allow_pickle=True) as npz:
            keys = list(npz.files)

            image_key = find_key(
                keys,
                ['image', 'images', 'input', 'inputs'],
            )
            subject_key = find_key(
                keys,
                ['subject_id', 'subject'],
            )
            channel_key = find_key(
                keys,
                ['input_columns', 'channel_names', 'channels'],
            )

            if image_key is None:
                raise KeyError(
                    f'No image array found. Available keys: {keys}'
                )

            image = np.asarray(npz[image_key])

            if image.ndim != 4:
                raise ValueError(
                    f'Expected a 4D CXYZ image, found {image.shape}.'
                )

            if subject_key is not None:
                case_result['subject_id'] = scalar_text(
                    npz[subject_key]
                )

            if channel_key is not None:
                channel_names = [
                    str(item)
                    for item in np.asarray(
                        npz[channel_key]
                    ).reshape(-1)
                ]
            elif image.shape[0] == len(EXPECTED_CHANNEL_NAMES):
                channel_names = EXPECTED_CHANNEL_NAMES
            else:
                channel_names = [
                    f'channel_{index}'
                    for index in range(image.shape[0])
                ]

            finite_mask = np.isfinite(image)
            finite_values = image[finite_mask]

            max_abs = (
                float(
                    np.max(
                        np.abs(
                            finite_values.astype(
                                np.float64,
                                copy=False,
                            )
                        )
                    )
                )
                if finite_values.size
                else np.nan
            )

            case_result.update({
                'readable': True,
                'image_key': image_key,
                'image_shape': str(tuple(image.shape)),
                'image_dtype': str(image.dtype),
                'image_channels': int(image.shape[0]),
                'image_nonfinite_voxels': int(
                    (~finite_mask).sum()
                ),
                'maximum_absolute_finite_value': max_abs,
            })

            schema_rows.append({
                'subject_id': case_result['subject_id'],
                'path': str(npz_path),
                'keys': json.dumps(keys),
                'channel_names': json.dumps(channel_names),
            })

            for channel_index in range(image.shape[0]):
                channel = np.asarray(
                    image[channel_index],
                    dtype=np.float32,
                )
                channel_finite = np.isfinite(channel)
                channel_values = channel[channel_finite]

                if channel_values.size:
                    percentiles = sampled_percentiles(
                        channel_values,
                        [0.1, 1, 50, 99, 99.9],
                    )
                    channel_min = float(
                        np.min(channel_values).astype(np.float64)
                    )
                    channel_max = float(
                        np.max(channel_values).astype(np.float64)
                    )
                    channel_max_abs = float(
                        np.max(
                            np.abs(
                                channel_values.astype(
                                    np.float64,
                                    copy=False,
                                )
                            )
                        )
                    )
                else:
                    percentiles = [np.nan] * 5
                    channel_min = np.nan
                    channel_max = np.nan
                    channel_max_abs = np.nan

                channel_name = (
                    channel_names[channel_index]
                    if channel_index < len(channel_names)
                    else f'channel_{channel_index}'
                )

                npz_channel_rows.append({
                    'subject_id': case_result['subject_id'],
                    'path': str(npz_path),
                    'channel_index': channel_index,
                    'channel_name': channel_name,
                    'shape': str(tuple(channel.shape)),
                    'dtype': str(channel.dtype),
                    'nonfinite_voxels': int(
                        (~channel_finite).sum()
                    ),
                    'finite_min': channel_min,
                    'finite_p001': percentiles[0],
                    'finite_p01': percentiles[1],
                    'finite_p50': percentiles[2],
                    'finite_p99': percentiles[3],
                    'finite_p999': percentiles[4],
                    'finite_max': channel_max,
                    'maximum_absolute_finite_value': channel_max_abs,
                })

    except Exception as exc:
        case_result['error'] = repr(exc)

    npz_case_rows.append(case_result)

    if (
        file_index == 1
        or file_index % 20 == 0
        or file_index == len(npz_paths)
    ):
        print(
            f'Audited existing v2 outputs: '
            f'{file_index}/{len(npz_paths)} files'
        )

npz_case_df = pd.DataFrame(npz_case_rows)
npz_channel_df = pd.DataFrame(npz_channel_rows)
schema_df = pd.DataFrame(schema_rows)

npz_case_path = DIAGNOSTIC_DIR / 'existing_v2_npz_case_audit.csv'
npz_channel_path = (
    DIAGNOSTIC_DIR / 'existing_v2_npz_channel_audit.csv'
)
schema_path = DIAGNOSTIC_DIR / 'existing_v2_npz_schema.csv'

npz_case_df.to_csv(npz_case_path, index=False)
npz_channel_df.to_csv(npz_channel_path, index=False)
schema_df.to_csv(schema_path, index=False)

print('\nExisting v2 files with case-level problems:')
case_problem_df = npz_case_df[
    (~npz_case_df['readable'])
    | (npz_case_df['image_nonfinite_voxels'].fillna(0) > 0)
    | (npz_case_df['image_channels'].fillna(-1) != 15)
]

if len(case_problem_df):
    display(case_problem_df)
else:
    print('None detected among files that were saved.')

print('\nExisting v2 channel ranges:')
if len(npz_channel_df):
    display(
        npz_channel_df.groupby(
            ['channel_index', 'channel_name'],
            as_index=False,
        ).agg(
            cases=('subject_id', 'count'),
            cases_with_nonfinite=(
                'nonfinite_voxels',
                lambda values: int((values > 0).sum()),
            ),
            global_min=('finite_min', 'min'),
            minimum_p001=('finite_p001', 'min'),
            median_p50=('finite_p50', 'median'),
            maximum_p999=('finite_p999', 'max'),
            global_max=('finite_max', 'max'),
            maximum_absolute_value=(
                'maximum_absolute_finite_value',
                'max',
            ),
        )
    )

print('\nFirst detected NPZ schema:')
if len(schema_df):
    display(schema_df.head(1).T)

summary = {
    'manifest_subjects': int(len(source_df)),
    'existing_v2_npz_files': int(len(npz_paths)),
    'raw_records': int(len(raw_audit_df)),
    'raw_problem_records': int(len(raw_problem_df)),
    'raw_problem_subjects': sorted(
        raw_problem_df['subject_id'].unique().tolist()
    ),
    'saved_npz_case_problems': int(len(case_problem_df)),
    'diagnostic_complete': True,
}

summary_path = DIAGNOSTIC_DIR / 'numeric_diagnostic_summary.json'
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding='utf-8',
)

print('\n' + '=' * 80)
print('TRIAL8 NUMERICAL DIAGNOSTIC COMPLETE')
print('=' * 80)
print(f'Raw input audit:     {raw_audit_path}')
print(f'Existing NPZ audit:  {npz_case_path}')
print(f'Channel audit:       {npz_channel_path}')
print(f'Schema audit:        {schema_path}')
print(f'Summary:             {summary_path}')
print('\nNo preprocessing files were changed or deleted.')


In [ ]:
import gc
import json
import os
import random
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'nibabel',
    'scipy',
    'openpyxl',
])

import nibabel as nib
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display
from nibabel.processing import resample_from_to, resample_to_output
from scipy import ndimage
from scipy.spatial import cKDTree





drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v1_trial0.csv'
)

PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
TARGET_SPACING = (0.8, 0.8, 2.0)
SEED = 20240824

PREPROCESSING_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
)

SUBJECT_OUTPUT_DIR = (
    PREPROCESSING_ROOT
    / 'subjects_npz'
)

CONFIG_DIR = PROJECT_ROOT / 'configs'
METADATA_DIR = PROJECT_ROOT / 'metadata'

AUDIT_PATH = (
    PREPROCESSING_ROOT
    / 'preprocessing_audit.csv'
)

PROGRESS_PATH = (
    PREPROCESSING_ROOT
    / 'preprocessing_progress.csv'
)

CONFIG_PATH = (
    CONFIG_DIR
    / f'{PREPROCESSING_VERSION}_config.json'
)

OUTPUT_MANIFEST_PATH = (
    METADATA_DIR
    / 'model_manifest_v2_vascufate_r1.csv'
)

for output_directory in [
    PREPROCESSING_ROOT,
    SUBJECT_OUTPUT_DIR,
    CONFIG_DIR,
    METADATA_DIR,
]:
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

def assert_project_path(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()

    if (
        path != project_root
        and project_root not in path.parents
    ):
        raise ValueError(
            f'Unsafe path outside project root: {path}'
        )

for output_path in [
    PREPROCESSING_ROOT,
    SUBJECT_OUTPUT_DIR,
    CONFIG_PATH,
    AUDIT_PATH,
    PROGRESS_PATH,
    OUTPUT_MANIFEST_PATH,
]:
    assert_project_path(output_path)

random.seed(SEED)
np.random.seed(SEED)

if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        SOURCE_MANIFEST_PATH
    )

manifest_df = pd.read_csv(
    SOURCE_MANIFEST_PATH
)

if len(manifest_df) != 148:
    raise RuntimeError(
        f'Expected 148 source-manifest subjects; '
        f'found {len(manifest_df)}.'
    )

if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError(
        'Source manifest contains duplicate subject IDs.'
    )

SOURCE_COLUMNS = {
    'ncct': 'path_raw_ncct',
    'cta': 'path_deriv_cta',
    'cbf': 'path_deriv_cbf',
    'cbv': 'path_deriv_cbv',
    'mtt': 'path_deriv_mtt',
    'tmax': 'path_deriv_tmax',
    'cow': 'path_cow_mask',
    'lvo': 'path_lvo_mask',
    'target': 'path_final_infarct_mask',
}

required_columns = [
    'subject_id',
    'dev_split',
    'lesion_volume_bin',
    *SOURCE_COLUMNS.values(),
]

missing_columns = [
    column
    for column in required_columns
    if column not in manifest_df.columns
]

if missing_columns:
    raise ValueError(
        f'Missing required manifest columns: '
        f'{missing_columns}'
    )





INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

STRUCTURAL_CHANNEL_INDICES = [0, 1, 2]
PERFUSION_CHANNEL_INDICES = list(
    range(3, len(INPUT_CHANNELS))
)





MTT_VALID_MIN = 0.0
MTT_VALID_MAX = 120.0
MTT_MIN_INTERPOLATION_SUPPORT = 0.999

VESSEL_NAMES = [
    'BA',
    'R_PCA',
    'L_PCA',
    'R_ICA',
    'R_MCA',
    'L_ICA',
    'L_MCA',
    'R_Pcom',
    'L_Pcom',
    'Acom',
    'R_ACA',
    'L_ACA',
    'third_A2',
]

VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

preprocessing_config = {
    'preprocessing_version': PREPROCESSING_VERSION,
    'source_manifest': str(
        SOURCE_MANIFEST_PATH
    ),
    'target_orientation': 'RAS',
    'target_spacing_mm': list(
        TARGET_SPACING
    ),
    'input_channels': INPUT_CHANNELS,
    'structural_channel_indices': (
        STRUCTURAL_CHANNEL_INDICES
    ),
    'perfusion_channel_indices': (
        PERFUSION_CHANNEL_INDICES
    ),
    'image_storage_dtype': 'float16',
    'numeric_revision': 'r1_mtt_sentinel_safe',
    'mtt_valid_range_seconds': [
        MTT_VALID_MIN,
        MTT_VALID_MAX,
    ],
    'mtt_min_interpolation_support': (
        MTT_MIN_INTERPOLATION_SUPPORT
    ),
    'continuous_invalid_value_policy': (
        'mask before interpolation; normalized masked interpolation'
    ),
    'contralateral_arithmetic': (
        'masked NumPy operations with no evaluation outside valid pairs'
    ),
    'output_commit_policy': (
        'validate temporary NPZ before atomic os.replace'
    ),
    'target_binary_interpolation': (
        'nearest neighbor'
    ),
    'target_soft_interpolation': (
        'trilinear occupancy'
    ),
    'cow_label_interpolation': (
        'nearest neighbor'
    ),
    'cow_soft_union_interpolation': (
        'trilinear'
    ),
    'lvo_representation': (
        'native-resolution exponential '
        'distance proximity'
    ),
    'lvo_proximity_scale_mm': 10.0,
    'crop_uses_target': False,
    'crop_margin_mm': [
        16.0,
        16.0,
        10.0,
    ],
    'vessel_names': VESSEL_NAMES,
    'vessel_feature_names': (
        VESSEL_FEATURE_NAMES
    ),
    'seed': SEED,
    'flagged_microlesion': (
        'sub-stroke0171'
    ),
}

CONFIG_PATH.write_text(
    json.dumps(
        preprocessing_config,
        indent=2,
    ),
    encoding='utf-8',
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Manifest subjects: {len(manifest_df)}')
print(f'Preprocessing version: {PREPROCESSING_VERSION}')
print(f'Target spacing: {TARGET_SPACING}')
print(f'Image channels: {len(INPUT_CHANNELS)}')
print(f'Subject output directory: {SUBJECT_OUTPUT_DIR}')





def orientation_code(image):
    return ''.join(
        nib.aff2axcodes(image.affine)
    )

def canonical_image(path):
    image = nib.load(str(path))
    canonical = nib.as_closest_canonical(
        image
    )

    if orientation_code(canonical) != 'RAS':
        raise RuntimeError(
            f'Could not canonicalize to RAS: {path}'
        )

    return canonical

def image_array(image):
    return np.asarray(
        image.dataobj,
        dtype=np.float32,
    )

def voxel_volume_mm3(image):
    return float(
        abs(
            np.linalg.det(
                image.affine[:3, :3]
            )
        )
    )

def grids_match(first, second):
    return (
        tuple(first.shape)
        == tuple(second.shape)
        and np.allclose(
            first.affine,
            second.affine,
            atol=1e-4,
            rtol=1e-5,
        )
    )

def harmonize_image(
    image,
    reference,
    order,
    cval,
):
    if grids_match(image, reference):
        return image, False

    harmonized = resample_from_to(
        image,
        (
            reference.shape,
            reference.affine,
        ),
        order=order,
        mode='constant',
        cval=float(cval),
    )

    if not grids_match(
        harmonized,
        reference,
    ):
        raise RuntimeError(
            'Grid harmonization failed.'
        )

    return harmonized, True

def make_image(
    array,
    affine,
):
    return nib.Nifti1Image(
        np.asarray(
            array,
            dtype=np.float32,
        ),
        affine,
    )

def resample_array(
    array,
    source_affine,
    output_grid,
    order,
    cval=0.0,
):
    result = resample_from_to(
        make_image(
            array,
            source_affine,
        ),
        output_grid,
        order=order,
        mode='constant',
        cval=float(cval),
    )

    return np.asarray(
        result.dataobj,
        dtype=np.float32,
    )

def valid_mtt_mask(array):
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    return (
        np.isfinite(array)
        & (array >= MTT_VALID_MIN)
        & (array <= MTT_VALID_MAX)
    )

def masked_resample_array(
    array,
    valid,
    source_affine,
    output_grid,
    order=1,
    minimum_support=0.999,
    invalid_fill=0.0,
):
    

    array = np.asarray(
        array,
        dtype=np.float32,
    )
    valid = (
        np.asarray(valid, dtype=bool)
        & np.isfinite(array)
    )

    numerator_source = np.zeros_like(
        array,
        dtype=np.float32,
    )
    numerator_source[valid] = array[valid]

    numerator = resample_array(
        numerator_source,
        source_affine,
        output_grid,
        order=order,
        cval=0.0,
    )
    support = resample_array(
        valid.astype(np.float32),
        source_affine,
        output_grid,
        order=order,
        cval=0.0,
    )

    output_valid = (
        np.isfinite(numerator)
        & np.isfinite(support)
        & (support >= float(minimum_support))
    )

    output = np.full(
        numerator.shape,
        float(invalid_fill),
        dtype=np.float32,
    )

    np.divide(
        numerator,
        support,
        out=output,
        where=output_valid,
    )

    output[~output_valid] = float(
        invalid_fill
    )

    if not np.isfinite(output).all():
        raise RuntimeError(
            'Masked resampling produced nonfinite values.'
        )

    return (
        output.astype(np.float32),
        output_valid.astype(bool),
        support.astype(np.float32),
    )





def largest_component_overlapping(
    candidate,
    preferred_mask=None,
):
    candidate = np.asarray(
        candidate,
        dtype=bool,
    )

    labels, component_count = (
        ndimage.label(
            candidate,
            structure=(
                ndimage.generate_binary_structure(
                    3,
                    1,
                )
            ),
        )
    )

    if component_count == 0:
        return np.zeros_like(
            candidate,
            dtype=bool,
        )

    component_sizes = np.bincount(
        labels.ravel()
    )
    component_sizes[0] = 0

    selected_label = None

    if (
        preferred_mask is not None
        and np.asarray(
            preferred_mask,
            dtype=bool,
        ).any()
    ):
        preferred_mask = np.asarray(
            preferred_mask,
            dtype=bool,
        )

        overlap_counts = np.bincount(
            labels[preferred_mask].ravel(),
            minlength=component_count + 1,
        )
        overlap_counts[0] = 0

        if overlap_counts.max() > 0:
            selected_label = int(
                overlap_counts.argmax()
            )

    if selected_label is None:
        selected_label = int(
            component_sizes.argmax()
        )

    return labels == selected_label

def construct_perfusion_validity(
    ncct,
    cbf,
    cbv,
    mtt,
    tmax,
    mtt_valid=None,
):
    finite = (
        np.isfinite(ncct)
        & np.isfinite(cbf)
        & np.isfinite(cbv)
        & np.isfinite(mtt)
        & np.isfinite(tmax)
    )

    if mtt_valid is not None:
        finite &= np.asarray(
            mtt_valid,
            dtype=bool,
        )

    ct_soft_tissue = (
        (ncct > -100.0)
        & (ncct < 200.0)
    )

    perfusion_defined = (
        (tmax > -29.5)
        | (cbf > 0.0)
        | (cbv > 0.0)
        | (mtt > 0.0)
    )

    validity = (
        finite
        & ct_soft_tissue
        & perfusion_defined
    )

    structure = (
        ndimage.generate_binary_structure(
            3,
            1,
        )
    )

    validity = ndimage.binary_closing(
        validity,
        structure=structure,
        iterations=2,
    )

    for z_index in range(
        validity.shape[2]
    ):
        validity[:, :, z_index] = (
            ndimage.binary_fill_holes(
                validity[:, :, z_index]
            )
        )

    validity = (
        largest_component_overlapping(
            validity
        )
    )

    validity = ndimage.binary_dilation(
        validity,
        structure=structure,
        iterations=1,
    )

    if mtt_valid is not None:
        validity &= np.asarray(
            mtt_valid,
            dtype=bool,
        )

    return validity.astype(bool)

def construct_broad_crop_mask(
    ncct,
    perfusion_valid,
    cow_union,
    lvo_proximity,
):
    finite = np.isfinite(ncct)

    head_candidate = (
        finite
        & (ncct > -500.0)
        & (ncct < 3000.0)
    )

    structure = (
        ndimage.generate_binary_structure(
            3,
            1,
        )
    )

    head_candidate = (
        ndimage.binary_closing(
            head_candidate,
            structure=structure,
            iterations=3,
        )
    )

    for z_index in range(
        head_candidate.shape[2]
    ):
        head_candidate[:, :, z_index] = (
            ndimage.binary_fill_holes(
                head_candidate[:, :, z_index]
            )
        )

    head_mask = (
        largest_component_overlapping(
            head_candidate,
            preferred_mask=perfusion_valid,
        )
    )

    head_mask = ndimage.binary_dilation(
        head_mask,
        structure=structure,
        iterations=2,
    )

    crop_mask = (
        head_mask
        | perfusion_valid
        | (cow_union > 0.01)
        | (lvo_proximity > 0.01)
    )

    crop_mask = ndimage.binary_closing(
        crop_mask,
        structure=structure,
        iterations=2,
    )

    return crop_mask.astype(bool)

def bounding_box_with_margin(
    mask,
    spacing,
    margin_mm=(16.0, 16.0, 10.0),
):
    coordinates = np.argwhere(mask)

    if len(coordinates) == 0:
        raise RuntimeError(
            'Crop mask is empty.'
        )

    minimum = coordinates.min(axis=0)
    maximum = coordinates.max(axis=0) + 1

    margin_voxels = np.ceil(
        np.asarray(margin_mm)
        / np.asarray(spacing)
    ).astype(int)

    minimum = np.maximum(
        minimum - margin_voxels,
        0,
    )

    maximum = np.minimum(
        maximum + margin_voxels,
        np.asarray(mask.shape),
    )

    return tuple(
        slice(
            int(minimum[axis]),
            int(maximum[axis]),
        )
        for axis in range(3)
    )

def cropped_affine(
    full_affine,
    crop_slices,
):
    crop_origin = np.asarray([
        crop_slices[0].start,
        crop_slices[1].start,
        crop_slices[2].start,
    ])

    output_affine = full_affine.copy()

    output_affine[:3, 3] = (
        nib.affines.apply_affine(
            full_affine,
            crop_origin,
        )
    )

    return output_affine

def mask_coverage(mask, container):
    mask = np.asarray(mask, dtype=bool)
    container = np.asarray(
        container,
        dtype=bool,
    )

    if not mask.any():
        return np.nan

    return float(
        np.logical_and(
            mask,
            container,
        ).sum()
        / mask.sum()
    )





def scaled_window(
    array,
    minimum,
    maximum,
    valid=None,
):
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    finite = np.isfinite(array)

    if valid is None:
        output_valid = finite
    else:
        output_valid = (
            np.asarray(valid, dtype=bool)
            & finite
        )

    safe_array = np.full_like(
        array,
        float(minimum),
        dtype=np.float32,
    )
    safe_array[output_valid] = (
        array[output_valid]
    )

    output = np.clip(
        safe_array,
        minimum,
        maximum,
    )

    output = (
        output - minimum
    ) / (
        maximum - minimum
    )

    output[~output_valid] = 0.0

    if not np.isfinite(output).all():
        raise RuntimeError(
            'Window scaling produced nonfinite values.'
        )

    return output.astype(np.float32)

def mirror_about_midline(
    array,
    midline_index,
):
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    x_indices = np.arange(
        array.shape[0]
    )

    mirrored_indices = np.rint(
        2.0 * midline_index
        - x_indices
    ).astype(int)

    index_valid = (
        (mirrored_indices >= 0)
        & (
            mirrored_indices
            < array.shape[0]
        )
    )

    output = np.zeros_like(
        array,
        dtype=np.float32,
    )

    output[index_valid] = (
        array[mirrored_indices[index_valid]]
    )

    return output, index_valid

def normalized_delta(
    value,
    mirror_value,
    scale,
    valid_pair,
):
    value = np.asarray(
        value,
        dtype=np.float32,
    )
    mirror_value = np.asarray(
        mirror_value,
        dtype=np.float32,
    )
    valid_pair = (
        np.asarray(valid_pair, dtype=bool)
        & np.isfinite(value)
        & np.isfinite(mirror_value)
    )

    delta = np.zeros_like(
        value,
        dtype=np.float32,
    )

    np.subtract(
        value,
        mirror_value,
        out=delta,
        where=valid_pair,
    )
    delta[valid_pair] /= float(scale)

    delta = np.clip(
        delta,
        -1.0,
        1.0,
    )

    delta[~valid_pair] = 0.0

    if not np.isfinite(delta).all():
        raise RuntimeError(
            'Normalized delta produced nonfinite values.'
        )

    return delta.astype(np.float32)

def relative_ratio(
    value,
    mirror_value,
    denominator_floor,
    valid_pair,
):
    value = np.asarray(
        value,
        dtype=np.float32,
    )
    mirror_value = np.asarray(
        mirror_value,
        dtype=np.float32,
    )

    denominator = (
        mirror_value
        + float(denominator_floor)
    )

    valid_pair = (
        np.asarray(valid_pair, dtype=bool)
        & np.isfinite(value)
        & np.isfinite(denominator)
        & (denominator > 0.0)
    )

    numerator = (
        value
        + float(denominator_floor)
    )

    ratio = np.zeros_like(
        value,
        dtype=np.float32,
    )

    np.divide(
        numerator,
        denominator,
        out=ratio,
        where=valid_pair,
    )

    ratio = np.clip(
        ratio,
        0.0,
        2.0,
    ) / 2.0

    ratio[~valid_pair] = 0.0

    if not np.isfinite(ratio).all():
        raise RuntimeError(
            'Relative ratio produced nonfinite values.'
        )

    return ratio.astype(np.float32)





def world_coordinates(
    voxel_coordinates,
    affine,
):
    return nib.affines.apply_affine(
        affine,
        voxel_coordinates,
    )

def extract_vessel_features(
    cow_labels,
    lvo_mask,
    affine,
    voxel_volume,
    brain_center_world,
    cta,
    cbf,
    cbv,
    mtt,
    tmax,
    perfusion_valid,
):
    feature_matrix = np.zeros(
        (
            len(VESSEL_NAMES),
            len(VESSEL_FEATURE_NAMES),
        ),
        dtype=np.float32,
    )

    presence = np.zeros(
        len(VESSEL_NAMES),
        dtype=np.uint8,
    )

    lvo_coordinates = np.argwhere(
        lvo_mask
    )

    if len(lvo_coordinates):
        lvo_world = world_coordinates(
            lvo_coordinates,
            affine,
        )
        lvo_tree = cKDTree(lvo_world)
        lvo_present = True
    else:
        lvo_tree = None
        lvo_present = False

    zooms = np.sqrt(
        np.sum(
            affine[:3, :3] ** 2,
            axis=0,
        )
    )

    modality_arrays = [
        cta,
        cbf,
        cbv,
        mtt,
        tmax,
    ]

    for vessel_index, vessel_name in enumerate(
        VESSEL_NAMES,
        start=1,
    ):
        vessel_mask = (
            cow_labels == vessel_index
        )

        vessel_coordinates = np.argwhere(
            vessel_mask
        )

        if len(vessel_coordinates) == 0:
            continue

        matrix_index = vessel_index - 1
        presence[matrix_index] = 1

        centroid_voxel = (
            vessel_coordinates.mean(axis=0)
        )

        centroid_world = (
            world_coordinates(
                centroid_voxel,
                affine,
            )
        )

        minimum_voxel = (
            vessel_coordinates.min(axis=0)
        )
        maximum_voxel = (
            vessel_coordinates.max(axis=0)
        )

        extent_mm = (
            maximum_voxel
            - minimum_voxel
            + 1
        ) * zooms

        cta_values = cta[vessel_mask]
        cta_values = cta_values[
            np.isfinite(cta_values)
        ].astype(
            np.float64,
            copy=False,
        )

        if len(cta_values) == 0:
            cta_mean = 0.0
            cta_p90 = 0.0
        else:
            cta_mean = float(
                np.mean(
                    cta_values,
                    dtype=np.float64,
                )
            )
            cta_p90 = float(
                np.percentile(
                    cta_values,
                    90,
                )
            )

        perfusion_means = []

        perfusion_measurement_mask = (
            vessel_mask
            & np.asarray(
                perfusion_valid,
                dtype=bool,
            )
        )

        for modality_array in modality_arrays[1:]:
            values = modality_array[
                perfusion_measurement_mask
            ]
            values = values[
                np.isfinite(values)
            ].astype(
                np.float64,
                copy=False,
            )

            if len(values):
                perfusion_means.append(
                    float(
                        np.mean(
                            values,
                            dtype=np.float64,
                        )
                    )
                )
            else:
                perfusion_means.append(0.0)

        if lvo_present:
            vessel_world = world_coordinates(
                vessel_coordinates,
                affine,
            )

            minimum_lvo_distance = float(
                lvo_tree.query(
                    vessel_world,
                    k=1,
                )[0].min()
            )
        else:
            minimum_lvo_distance = 100.0

        overlap_fraction = float(
            np.logical_and(
                vessel_mask,
                lvo_mask,
            ).sum()
            / vessel_mask.sum()
        )

        feature_matrix[matrix_index] = [
            1.0,
            float(
                vessel_mask.sum()
                * voxel_volume
                / 1000.0
            ),
            float(
                centroid_world[0]
                - brain_center_world[0]
            ),
            float(
                centroid_world[1]
                - brain_center_world[1]
            ),
            float(
                centroid_world[2]
                - brain_center_world[2]
            ),
            float(extent_mm[0]),
            float(extent_mm[1]),
            float(extent_mm[2]),
            cta_mean,
            cta_p90,
            perfusion_means[0],
            perfusion_means[1],
            perfusion_means[2],
            perfusion_means[3],
            minimum_lvo_distance,
            overlap_fraction,
        ]

    return (
        feature_matrix,
        presence,
        int(lvo_present),
    )





REQUIRED_NPZ_KEYS = {
    'image',
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'full_resampled_affine',
    'crop_origin',
    'full_resampled_shape',
    'subject_id',
    'preprocessing_version',
    'source_mtt_invalid_voxels',
    'source_mtt_valid_fraction',
}

def validate_output_npz(path):
    result = {
        'valid': False,
        'error': None,
    }

    try:
        with np.load(
            path,
            allow_pickle=False,
        ) as data:
            missing_keys = (
                REQUIRED_NPZ_KEYS
                - set(data.files)
            )

            if missing_keys:
                raise ValueError(
                    f'Missing NPZ keys: '
                    f'{sorted(missing_keys)}'
                )

            image = data['image']
            target_binary = (
                data['target_binary']
            )
            target_soft = (
                data['target_soft']
            )
            cow_label = data['cow_label']
            crop_mask = data['crop_mask']
            perfusion_valid = (
                data['perfusion_valid']
            )
            mtt_valid = data['mtt_valid']
            mtt_support = (
                data['mtt_interpolation_support']
            )
            cow_soft_union = (
                data['cow_soft_union']
            )
            lvo_proximity = (
                data['lvo_proximity']
            )
            vessel_features = (
                data['vessel_features_raw']
            )

            spatial_shape = tuple(
                image.shape[1:]
            )

            if image.shape[0] != len(
                INPUT_CHANNELS
            ):
                raise ValueError(
                    f'Expected '
                    f'{len(INPUT_CHANNELS)} channels; '
                    f'found {image.shape[0]}.'
                )

            for name, array in [
                (
                    'target_binary',
                    target_binary,
                ),
                ('target_soft', target_soft),
                ('crop_mask', crop_mask),
                (
                    'perfusion_valid',
                    perfusion_valid,
                ),
                ('mtt_valid', mtt_valid),
                (
                    'mtt_interpolation_support',
                    mtt_support,
                ),
                ('cow_label', cow_label),
                (
                    'cow_soft_union',
                    cow_soft_union,
                ),
                (
                    'lvo_proximity',
                    lvo_proximity,
                ),
            ]:
                if tuple(array.shape) != spatial_shape:
                    raise ValueError(
                        f'{name} shape mismatch: '
                        f'{array.shape} vs '
                        f'{spatial_shape}'
                    )

            if not np.isfinite(image).all():
                raise ValueError(
                    'Image contains nonfinite values.'
                )

            if not np.isfinite(
                target_soft
            ).all():
                raise ValueError(
                    'Soft target contains '
                    'nonfinite values.'
                )

            if not np.isfinite(
                lvo_proximity
            ).all():
                raise ValueError(
                    'LVO proximity contains '
                    'nonfinite values.'
                )

            if not np.isfinite(
                vessel_features
            ).all():
                raise ValueError(
                    'Vessel features contain '
                    'nonfinite values.'
                )

            if not np.isfinite(
                mtt_support
            ).all():
                raise ValueError(
                    'MTT interpolation support contains '
                    'nonfinite values.'
                )

            if (
                mtt_support.min() < -0.0001
                or mtt_support.max() > 1.0001
            ):
                raise ValueError(
                    'MTT interpolation support is outside '
                    '[0, 1].'
                )

            if vessel_features.shape != (
                len(VESSEL_NAMES),
                len(VESSEL_FEATURE_NAMES),
            ):
                raise ValueError(
                    'Vessel feature shape is invalid: '
                    f'{vessel_features.shape}.'
                )

            if image.min() < -1.0001:
                raise ValueError(
                    'Image minimum is below -1.'
                )

            if image.max() > 1.0001:
                raise ValueError(
                    'Image maximum is above 1.'
                )

            non_delta_indices = [
                index
                for index in range(
                    len(INPUT_CHANNELS)
                )
                if index not in {9, 10, 11, 12}
            ]

            if image[non_delta_indices].min() < -0.0001:
                raise ValueError(
                    'A non-delta image channel is below zero.'
                )

            stored_channels = [
                str(value)
                for value in data[
                    'input_channels'
                ].tolist()
            ]

            if stored_channels != INPUT_CHANNELS:
                raise ValueError(
                    'Stored input-channel order does not '
                    'match the configured order.'
                )

            stored_version = str(
                data[
                    'preprocessing_version'
                ].item()
            )

            if stored_version != PREPROCESSING_VERSION:
                raise ValueError(
                    'Stored preprocessing version does '
                    f'not match {PREPROCESSING_VERSION}: '
                    f'{stored_version}.'
                )

            if (
                target_binary.min() < 0
                or target_binary.max() > 1
            ):
                raise ValueError(
                    'Binary target is invalid.'
                )

            result.update({
                'valid': True,
                'image_shape': tuple(
                    int(value)
                    for value in image.shape
                ),
                'target_binary_voxels': int(
                    target_binary.sum()
                ),
                'target_soft_sum': float(
                    target_soft.sum()
                ),
                'lvo_present': int(
                    data['lvo_present']
                ),
                'vessels_present': int(
                    data[
                        'vessel_presence'
                    ].sum()
                ),
                'binary_loss_weight': float(
                    data[
                        'binary_loss_weight'
                    ]
                ),
                'source_mtt_invalid_voxels': int(
                    data[
                        'source_mtt_invalid_voxels'
                    ]
                ),
                'source_mtt_valid_fraction': float(
                    data[
                        'source_mtt_valid_fraction'
                    ]
                ),
            })

    except Exception as exc:
        result['error'] = repr(exc)

    return result





def process_subject(row, output_path):
    subject_id = row['subject_id']

    source_images = {
        role: canonical_image(
            row[column]
        )
        for role, column
        in SOURCE_COLUMNS.items()
    }

    raw_mtt_source = image_array(
        source_images['mtt']
    )
    source_mtt_valid = valid_mtt_mask(
        raw_mtt_source
    )
    source_mtt_invalid_voxels = int(
        (~source_mtt_valid).sum()
    )

    sanitized_mtt_source = np.zeros_like(
        raw_mtt_source,
        dtype=np.float32,
    )
    sanitized_mtt_source[
        source_mtt_valid
    ] = raw_mtt_source[source_mtt_valid]

    mtt_source_image = make_image(
        sanitized_mtt_source,
        source_images['mtt'].affine,
    )
    mtt_valid_source_image = make_image(
        source_mtt_valid.astype(np.float32),
        source_images['mtt'].affine,
    )

    reference_native = (
        source_images['ncct']
    )

    harmonized_images = {
        'ncct': reference_native
    }

    harmonized_roles = []
    native_mtt_valid = None

    for role in [
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'cow',
        'lvo',
        'target',
    ]:
        if role == 'mtt':
            if grids_match(
                mtt_source_image,
                reference_native,
            ):
                harmonized = mtt_source_image
                native_mtt_valid = (
                    image_array(
                        mtt_valid_source_image
                    ) > 0.5
                )
                was_resampled = False
            else:
                (
                    harmonized_mtt_array,
                    native_mtt_valid,
                    _,
                ) = masked_resample_array(
                    array=sanitized_mtt_source,
                    valid=source_mtt_valid,
                    source_affine=(
                        source_images['mtt'].affine
                    ),
                    output_grid=(
                        reference_native.shape,
                        reference_native.affine,
                    ),
                    order=1,
                    minimum_support=(
                        MTT_MIN_INTERPOLATION_SUPPORT
                    ),
                    invalid_fill=0.0,
                )
                harmonized = make_image(
                    harmonized_mtt_array,
                    reference_native.affine,
                )
                was_resampled = True

            harmonized_images[role] = harmonized

            if was_resampled:
                harmonized_roles.append(role)

            continue

        order = (
            1
            if role in {
                'cta',
                'cbf',
                'cbv',
                'mtt',
                'tmax',
            }
            else 0
        )

        cval = {
            'cta': -1000.0,
            'cbf': 0.0,
            'cbv': 0.0,
            'mtt': 0.0,
            'tmax': -30.0,
            'cow': 0.0,
            'lvo': 0.0,
            'target': 0.0,
        }[role]

        harmonized, was_resampled = (
            harmonize_image(
                source_images[role],
                reference_native,
                order=order,
                cval=cval,
            )
        )

        harmonized_images[role] = (
            harmonized
        )

        if was_resampled:
            harmonized_roles.append(role)

    native_arrays = {
        role: image_array(image)
        for role, image
        in harmonized_images.items()
    }

    ncct_native = native_arrays['ncct']
    cta_native = native_arrays['cta']
    cbf_native = native_arrays['cbf']
    cbv_native = native_arrays['cbv']
    mtt_native = native_arrays['mtt']
    tmax_native = native_arrays['tmax']

    cow_native = np.rint(
        native_arrays['cow']
    ).astype(np.int16)

    lvo_native = (
        native_arrays['lvo'] > 0.5
    )

    target_native = (
        native_arrays['target'] > 0.5
    )

    target_source_array = (
        image_array(
            source_images['target']
        ) > 0
    )

    source_target_volume_ml = float(
        target_source_array.sum()
        * voxel_volume_mm3(
            source_images['target']
        )
        / 1000.0
    )

    native_perfusion_valid = (
        construct_perfusion_validity(
            ncct=ncct_native,
            cbf=cbf_native,
            cbv=cbv_native,
            mtt=mtt_native,
            tmax=tmax_native,
            mtt_valid=native_mtt_valid,
        )
    )

    native_valid_coordinates = (
        np.argwhere(
            native_perfusion_valid
        )
    )

    if len(native_valid_coordinates):
        native_center_voxel = (
            native_valid_coordinates.min(axis=0)
            + native_valid_coordinates.max(
                axis=0
            )
        ) / 2.0
    else:
        native_center_voxel = (
            np.asarray(
                reference_native.shape
            ) - 1
        ) / 2.0

    brain_center_world = (
        nib.affines.apply_affine(
            reference_native.affine,
            native_center_voxel,
        )
    )

    (
        vessel_features,
        vessel_presence,
        lvo_present,
    ) = extract_vessel_features(
        cow_labels=cow_native,
        lvo_mask=lvo_native,
        affine=reference_native.affine,
        voxel_volume=voxel_volume_mm3(
            reference_native
        ),
        brain_center_world=(
            brain_center_world
        ),
        cta=cta_native,
        cbf=cbf_native,
        cbv=cbv_native,
        mtt=mtt_native,
        tmax=tmax_native,
        perfusion_valid=(
            native_perfusion_valid
        ),
    )

    if lvo_present:
        native_spacing = (
            reference_native.header
            .get_zooms()[:3]
        )

        native_lvo_distance = (
            ndimage.distance_transform_edt(
                ~lvo_native,
                sampling=native_spacing,
            )
        ).astype(np.float32)

        native_lvo_proximity = np.exp(
            -native_lvo_distance / 10.0
        ).astype(np.float32)
    else:
        native_lvo_proximity = (
            np.zeros_like(
                ncct_native,
                dtype=np.float32,
            )
        )

    output_reference = (
        resample_to_output(
            reference_native,
            voxel_sizes=TARGET_SPACING,
            order=1,
            mode='constant',
            cval=-1000.0,
        )
    )

    output_grid = (
        output_reference.shape,
        output_reference.affine,
    )

    ncct = image_array(
        output_reference
    )

    cta = resample_array(
        cta_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=-1000.0,
    )

    cbf = resample_array(
        cbf_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=0.0,
    )

    cbv = resample_array(
        cbv_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=0.0,
    )

    (
        mtt,
        output_mtt_valid,
        output_mtt_support,
    ) = masked_resample_array(
        array=mtt_native,
        valid=native_mtt_valid,
        source_affine=(
            reference_native.affine
        ),
        output_grid=output_grid,
        order=1,
        minimum_support=(
            MTT_MIN_INTERPOLATION_SUPPORT
        ),
        invalid_fill=0.0,
    )

    tmax = resample_array(
        tmax_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=-30.0,
    )

    target_binary = (
        resample_array(
            target_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        ) > 0.5
    )

    target_soft = np.clip(
        resample_array(
            target_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    cow_label = np.rint(
        resample_array(
            cow_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        )
    ).astype(np.uint8)

    cow_soft_union = np.clip(
        resample_array(
            (cow_native > 0).astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    lvo_proximity = np.clip(
        resample_array(
            native_lvo_proximity,
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    perfusion_valid = (
        resample_array(
            native_perfusion_valid.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        ) > 0.5
    )

    perfusion_valid &= output_mtt_valid

    crop_mask_full = (
        construct_broad_crop_mask(
            ncct=ncct,
            perfusion_valid=(
                perfusion_valid
            ),
            cow_union=cow_soft_union,
            lvo_proximity=lvo_proximity,
        )
    )

    crop_slices = (
        bounding_box_with_margin(
            crop_mask_full,
            spacing=TARGET_SPACING,
            margin_mm=(
                16.0,
                16.0,
                10.0,
            ),
        )
    )

    crop_origin = np.asarray([
        crop_slices[0].start,
        crop_slices[1].start,
        crop_slices[2].start,
    ], dtype=np.int32)

    crop_mask = (
        crop_mask_full[crop_slices]
    )

    perfusion_valid_crop = (
        perfusion_valid[crop_slices]
    )

    mtt_valid_crop = (
        output_mtt_valid[crop_slices]
    )

    mtt_support_crop = (
        output_mtt_support[crop_slices]
    )

    target_binary_crop = (
        target_binary[crop_slices]
    )

    target_soft_crop = (
        target_soft[crop_slices]
    )

    cow_label_crop = (
        cow_label[crop_slices]
    )

    cow_soft_union_crop = (
        cow_soft_union[crop_slices]
    )

    lvo_proximity_crop = (
        lvo_proximity[crop_slices]
    )

    
    crop_coordinates = np.argwhere(
        crop_mask_full
    )

    midline_index = float(
        (
            crop_coordinates[:, 0].min()
            + crop_coordinates[:, 0].max()
        ) / 2.0
    )

    mirror_cbf, mirror_index_valid = (
        mirror_about_midline(
            cbf,
            midline_index,
        )
    )

    mirror_cbv, _ = mirror_about_midline(
        cbv,
        midline_index,
    )

    mirror_mtt, _ = mirror_about_midline(
        mtt,
        midline_index,
    )

    mirror_tmax, _ = mirror_about_midline(
        tmax,
        midline_index,
    )

    mirror_valid_float, _ = (
        mirror_about_midline(
            perfusion_valid.astype(
                np.float32
            ),
            midline_index,
        )
    )

    mirror_valid = (
        mirror_valid_float > 0.5
    )

    index_valid_volume = np.broadcast_to(
        mirror_index_valid[
            :,
            None,
            None,
        ],
        perfusion_valid.shape,
    )

    pair_valid = (
        perfusion_valid
        & mirror_valid
        & index_valid_volume
        & output_mtt_valid
        & np.isfinite(cbf)
        & np.isfinite(cbv)
        & np.isfinite(mtt)
        & np.isfinite(tmax)
        & np.isfinite(mirror_cbf)
        & np.isfinite(mirror_cbv)
        & np.isfinite(mirror_mtt)
        & np.isfinite(mirror_tmax)
    )

    relative_cbf = relative_ratio(
        cbf,
        mirror_cbf,
        denominator_floor=5.0,
        valid_pair=pair_valid,
    )

    relative_cbv = relative_ratio(
        cbv,
        mirror_cbv,
        denominator_floor=0.5,
        valid_pair=pair_valid,
    )

    delta_cbf = normalized_delta(
        cbf,
        mirror_cbf,
        scale=100.0,
        valid_pair=pair_valid,
    )

    delta_cbv = normalized_delta(
        cbv,
        mirror_cbv,
        scale=10.0,
        valid_pair=pair_valid,
    )

    delta_mtt = normalized_delta(
        mtt,
        mirror_mtt,
        scale=20.0,
        valid_pair=pair_valid,
    )

    delta_tmax = normalized_delta(
        tmax,
        mirror_tmax,
        scale=15.0,
        valid_pair=pair_valid,
    )

    channel_arrays = [
        scaled_window(
            ncct,
            -20.0,
            100.0,
        )[crop_slices],
        scaled_window(
            ncct,
            -100.0,
            300.0,
        )[crop_slices],
        scaled_window(
            cta,
            -100.0,
            1000.0,
        )[crop_slices],
        scaled_window(
            cbf,
            0.0,
            150.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            cbv,
            0.0,
            10.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            mtt,
            0.0,
            30.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            tmax,
            0.0,
            20.0,
            valid=perfusion_valid,
        )[crop_slices],
        relative_cbf[crop_slices],
        relative_cbv[crop_slices],
        delta_cbf[crop_slices],
        delta_cbv[crop_slices],
        delta_mtt[crop_slices],
        delta_tmax[crop_slices],
        perfusion_valid_crop.astype(
            np.float32
        ),
        pair_valid[crop_slices].astype(
            np.float32
        ),
    ]

    for channel_name, channel_array in zip(
        INPUT_CHANNELS,
        channel_arrays,
    ):
        if not np.isfinite(
            channel_array
        ).all():
            raise RuntimeError(
                f'Channel {channel_name} contains '
                'nonfinite values before storage.'
            )

        if channel_name.startswith('delta_'):
            if (
                channel_array.min() < -1.0001
                or channel_array.max() > 1.0001
            ):
                raise RuntimeError(
                    f'Delta channel {channel_name} is '
                    'outside [-1, 1].'
                )
        elif (
            channel_array.min() < -0.0001
            or channel_array.max() > 1.0001
        ):
            raise RuntimeError(
                f'Channel {channel_name} is outside '
                '[0, 1].'
            )

    image = np.stack(
        channel_arrays,
        axis=0,
    ).astype(np.float16)

    output_affine = cropped_affine(
        output_reference.affine,
        crop_slices,
    )

    output_voxel_volume = (
        voxel_volume_mm3(
            output_reference
        )
    )

    target_binary_volume_ml = float(
        target_binary_crop.sum()
        * output_voxel_volume
        / 1000.0
    )

    target_soft_volume_ml = float(
        target_soft_crop.sum()
        * output_voxel_volume
        / 1000.0
    )

    source_target_nonempty = (
        source_target_volume_ml > 0
    )

    binary_target_nonempty = bool(
        target_binary_crop.any()
    )

    binary_loss_weight = 1.0

    if (
        source_target_nonempty
        and not binary_target_nonempty
    ):
        binary_loss_weight = 0.0

    target_crop_coverage = (
        mask_coverage(
            target_binary,
            crop_mask_full,
        )
    )

    full_resampled_shape = np.asarray(
        output_reference.shape,
        dtype=np.int32,
    )

    temporary_path = Path(
        str(output_path) + '.tmp'
    )

    with temporary_path.open('wb') as file:
        np.savez_compressed(
            file,
            image=image,
            target_binary=(
                target_binary_crop.astype(
                    np.uint8
                )
            ),
            target_soft=(
                target_soft_crop.astype(
                    np.float16
                )
            ),
            binary_loss_weight=np.asarray(
                binary_loss_weight,
                dtype=np.float32,
            ),
            crop_mask=(
                crop_mask.astype(
                    np.uint8
                )
            ),
            perfusion_valid=(
                perfusion_valid_crop.astype(
                    np.uint8
                )
            ),
            mtt_valid=(
                mtt_valid_crop.astype(
                    np.uint8
                )
            ),
            mtt_interpolation_support=(
                np.clip(
                    mtt_support_crop,
                    0.0,
                    1.0,
                ).astype(np.float16)
            ),
            cow_label=(
                cow_label_crop.astype(
                    np.uint8
                )
            ),
            cow_soft_union=(
                cow_soft_union_crop.astype(
                    np.float16
                )
            ),
            lvo_proximity=(
                lvo_proximity_crop.astype(
                    np.float16
                )
            ),
            lvo_present=np.asarray(
                lvo_present,
                dtype=np.uint8,
            ),
            vessel_features_raw=(
                vessel_features.astype(
                    np.float32
                )
            ),
            vessel_presence=(
                vessel_presence.astype(
                    np.uint8
                )
            ),
            input_channels=np.asarray(
                INPUT_CHANNELS
            ),
            structural_channel_indices=(
                np.asarray(
                    STRUCTURAL_CHANNEL_INDICES,
                    dtype=np.int16,
                )
            ),
            perfusion_channel_indices=(
                np.asarray(
                    PERFUSION_CHANNEL_INDICES,
                    dtype=np.int16,
                )
            ),
            vessel_names=np.asarray(
                VESSEL_NAMES
            ),
            vessel_feature_names=np.asarray(
                VESSEL_FEATURE_NAMES
            ),
            affine=output_affine.astype(
                np.float64
            ),
            full_resampled_affine=(
                output_reference.affine.astype(
                    np.float64
                )
            ),
            crop_origin=crop_origin,
            full_resampled_shape=(
                full_resampled_shape
            ),
            target_spacing=np.asarray(
                TARGET_SPACING,
                dtype=np.float32,
            ),
            subject_id=np.asarray(
                subject_id
            ),
            preprocessing_version=np.asarray(
                PREPROCESSING_VERSION
            ),
            source_target_volume_ml=np.asarray(
                source_target_volume_ml,
                dtype=np.float32,
            ),
            target_binary_volume_ml=np.asarray(
                target_binary_volume_ml,
                dtype=np.float32,
            ),
            target_soft_volume_ml=np.asarray(
                target_soft_volume_ml,
                dtype=np.float32,
            ),
            target_crop_coverage=np.asarray(
                target_crop_coverage,
                dtype=np.float32,
            ),
            estimated_midline_full_index=np.asarray(
                midline_index,
                dtype=np.float32,
            ),
            harmonized_roles=np.asarray(
                harmonized_roles
            ),
            source_mtt_invalid_voxels=np.asarray(
                source_mtt_invalid_voxels,
                dtype=np.int64,
            ),
            source_mtt_valid_fraction=np.asarray(
                source_mtt_valid.mean(
                    dtype=np.float64
                ),
                dtype=np.float32,
            ),
        )

    result = validate_output_npz(
        temporary_path
    )

    if not result['valid']:
        temporary_path.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            f'Temporary output validation failed: '
            f'{result["error"]}'
        )

    os.replace(
        temporary_path,
        output_path,
    )

    committed_result = validate_output_npz(
        output_path
    )

    if not committed_result['valid']:
        raise RuntimeError(
            'Committed output failed validation: '
            f'{committed_result["error"]}'
        )

    result = committed_result

    result.update({
        'subject_id': subject_id,
        'output_path': str(output_path),
        'source_target_volume_ml': (
            source_target_volume_ml
        ),
        'target_binary_volume_ml': (
            target_binary_volume_ml
        ),
        'target_soft_volume_ml': (
            target_soft_volume_ml
        ),
        'target_crop_coverage': (
            target_crop_coverage
        ),
        'crop_shape': tuple(
            int(value)
            for value in image.shape[1:]
        ),
        'harmonized_roles': (
            harmonized_roles
        ),
        'source_mtt_invalid_voxels': (
            source_mtt_invalid_voxels
        ),
        'status': 'processed',
    })

    del (
        source_images,
        harmonized_images,
        native_arrays,
        ncct_native,
        cta_native,
        cbf_native,
        cbv_native,
        mtt_native,
        tmax_native,
        cow_native,
        lvo_native,
        target_native,
        target_source_array,
        native_perfusion_valid,
        native_lvo_proximity,
        output_reference,
        ncct,
        cta,
        cbf,
        cbv,
        mtt,
        tmax,
        target_binary,
        target_soft,
        cow_label,
        cow_soft_union,
        lvo_proximity,
        perfusion_valid,
        crop_mask_full,
        mirror_cbf,
        mirror_cbv,
        mirror_mtt,
        mirror_tmax,
        mirror_valid_float,
        mirror_valid,
        pair_valid,
        relative_cbf,
        relative_cbv,
        delta_cbf,
        delta_cbv,
        delta_mtt,
        delta_tmax,
        channel_arrays,
        image,
    )

    gc.collect()

    return result





audit_rows = []

for subject_index, (_, row) in enumerate(
    manifest_df
    .sort_values('subject_id')
    .iterrows(),
    start=1,
):
    subject_id = row['subject_id']

    output_path = (
        SUBJECT_OUTPUT_DIR
        / (
            f'{subject_id}_'
            f'{PREPROCESSING_VERSION}.npz'
        )
    )

    print(
        f'\n{subject_index}/{len(manifest_df)} '
        f'{subject_id}'
    )

    audit_record = {
        'subject_id': subject_id,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': (
            row['lesion_volume_bin']
        ),
        'output_path': str(output_path),
        'status': None,
        'valid': False,
        'error': None,
    }

    try:
        if output_path.exists():
            existing_result = (
                validate_output_npz(
                    output_path
                )
            )

            if existing_result['valid']:
                audit_record.update(
                    existing_result
                )
                audit_record['status'] = (
                    'skipped_valid_existing'
                )

                print(
                    '  Valid existing output found. '
                    'Skipping.'
                )
            else:
                print(
                    '  Existing output is invalid. '
                    'Regenerating.'
                )

                processed_result = (
                    process_subject(
                        row,
                        output_path,
                    )
                )

                audit_record.update(
                    processed_result
                )
        else:
            processed_result = process_subject(
                row,
                output_path,
            )

            audit_record.update(
                processed_result
            )

        print(
            f'  Status: '
            f'{audit_record["status"]}'
        )
        print(
            f'  Image shape: '
            f'{audit_record.get("image_shape")}'
        )
        print(
            f'  Binary target voxels: '
            f'{audit_record.get("target_binary_voxels")}'
        )
        print(
            f'  Vessels present: '
            f'{audit_record.get("vessels_present")}'
        )

    except Exception as exc:
        audit_record.update({
            'status': 'failed',
            'valid': False,
            'error': repr(exc),
        })

        print(f'  FAILED: {exc}')

    audit_rows.append(
        audit_record
    )

    pd.DataFrame(
        audit_rows
    ).to_csv(
        PROGRESS_PATH,
        index=False,
    )

    gc.collect()





final_audit_rows = []

for _, row in (
    manifest_df
    .sort_values('subject_id')
    .iterrows()
):
    subject_id = row['subject_id']

    output_path = (
        SUBJECT_OUTPUT_DIR
        / (
            f'{subject_id}_'
            f'{PREPROCESSING_VERSION}.npz'
        )
    )

    validation = validate_output_npz(
        output_path
    ) if output_path.exists() else {
        'valid': False,
        'error': 'Output file missing.',
    }

    final_audit_rows.append({
        'subject_id': subject_id,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': (
            row['lesion_volume_bin']
        ),
        'output_path': str(output_path),
        **validation,
    })

audit_df = pd.DataFrame(
    final_audit_rows
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False,
)

valid_audit_df = audit_df[
    audit_df['valid']
].copy()

output_manifest_df = (
    manifest_df.merge(
        valid_audit_df[
            [
                'subject_id',
                'output_path',
                'image_shape',
                'target_binary_voxels',
                'target_soft_sum',
                'lvo_present',
                'vessels_present',
                'binary_loss_weight',
                'source_mtt_invalid_voxels',
                'source_mtt_valid_fraction',
            ]
        ],
        on='subject_id',
        how='inner',
        validate='one_to_one',
    )
)

output_manifest_df = (
    output_manifest_df.rename(
        columns={
            'output_path': (
                'vascufate_v2_npz_path'
            ),
            'image_shape': (
                'vascufate_v2_image_shape'
            ),
            'target_binary_voxels': (
                'vascufate_v2_target_binary_voxels'
            ),
            'target_soft_sum': (
                'vascufate_v2_target_soft_sum'
            ),
            'lvo_present': (
                'vascufate_v2_lvo_present'
            ),
            'vessels_present': (
                'vascufate_v2_vessels_present'
            ),
            'binary_loss_weight': (
                'vascufate_v2_binary_loss_weight'
            ),
            'source_mtt_invalid_voxels': (
                'vascufate_v2_source_mtt_invalid_voxels'
            ),
            'source_mtt_valid_fraction': (
                'vascufate_v2_source_mtt_valid_fraction'
            ),
        }
    )
)

expected_split_counts = {
    'train': 102,
    'val': 23,
    'test': 23,
}
observed_split_counts = (
    output_manifest_df[
        'dev_split'
    ].value_counts().to_dict()
)

if observed_split_counts != expected_split_counts:
    raise RuntimeError(
        'Final split-count gate failed. Expected '
        f'{expected_split_counts}, found '
        f'{observed_split_counts}. Do not continue.'
    )

output_manifest_df.to_csv(
    OUTPUT_MANIFEST_PATH,
    index=False,
)

print('\n' + '=' * 80)
print('TRIAL8 V2 CORE PREPROCESSING COMPLETE')
print('=' * 80)

print(f'Total expected subjects: {len(manifest_df)}')
print(f'Valid outputs: {len(valid_audit_df)}')
print(
    f'Failed or missing outputs: '
    f'{len(manifest_df) - len(valid_audit_df)}'
)

print('\nStatus counts:')
display(
    audit_df[
        'valid'
    ].value_counts(
        dropna=False
    ).rename_axis(
        'valid'
    ).reset_index(
        name='subjects'
    )
)

print('\nSplit counts in v2 manifest:')
display(
    output_manifest_df
    .groupby('dev_split')
    .size()
    .reset_index(name='subjects')
)

print('\nBinary loss-weight counts:')
display(
    output_manifest_df[
        'vascufate_v2_binary_loss_weight'
    ].value_counts(
        dropna=False
    ).rename_axis(
        'binary_loss_weight'
    ).reset_index(
        name='subjects'
    )
)

print('\nSubjects with empty binary targets:')
display(
    output_manifest_df[
        output_manifest_df[
            'vascufate_v2_target_binary_voxels'
        ] == 0
    ][
        [
            'subject_id',
            'dev_split',
            'lesion_volume_bin',
            'vascufate_v2_target_soft_sum',
            'vascufate_v2_binary_loss_weight',
        ]
    ]
)

print('\nInvalid outputs:')
display(
    audit_df[
        ~audit_df['valid']
    ][
        [
            'subject_id',
            'output_path',
            'error',
        ]
    ]
)

print(f'\nSaved configuration: {CONFIG_PATH}')
print(f'Saved audit: {AUDIT_PATH}')
print(f'Saved v2 manifest: {OUTPUT_MANIFEST_PATH}')
print(f'Subject outputs: {SUBJECT_OUTPUT_DIR}')

if len(valid_audit_df) != len(
    manifest_df
):
    raise RuntimeError(
        'Final preprocessing gate failed: expected '
        f'{len(manifest_df)} valid subjects but found '
        f'{len(valid_audit_df)}. Rerun this standalone '
        'cell to resume only missing or invalid r1 '
        'subjects. Do not continue to atlas processing.'
    )
else:
    print(
        '\nAll subjects completed and passed '
        'NPZ validation.'
    )

print(
    '\nNo Trial8 model has been trained. '
    'The vascular-territory atlas sidecars '
    'are the next stage.'
)


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'nibabel',
    'pandas',
])

import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v1_trial0.csv'
)

OLD_VERSION = 'v2_vascufate_ras_0p8x0p8x2'
NEW_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
EXPECTED_SUBJECTS = 148
EXPECTED_CHANNELS = 15
EXPECTED_SPACING = np.asarray((0.8, 0.8, 2.0), dtype=np.float64)
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}

OLD_SUBJECT_DIR = (
    PROJECT_ROOT / 'preprocessing' / OLD_VERSION / 'subjects_npz'
)
NEW_ROOT = PROJECT_ROOT / 'preprocessing' / NEW_VERSION
NEW_SUBJECT_DIR = NEW_ROOT / 'subjects_npz'
NEW_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1.csv'
)
AUDIT_DIR = NEW_ROOT / 'final_verification'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CASE_AUDIT_PATH = AUDIT_DIR / 'r1_case_audit.csv'
CHANNEL_AUDIT_PATH = AUDIT_DIR / 'r1_channel_audit.csv'
COMPARISON_PATH = AUDIT_DIR / 'r1_vs_original_geometry_audit.csv'
FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
GATE_PATH = AUDIT_DIR / 'r1_verification_gate.json'

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

REQUIRED_KEYS = {
    'image',
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'full_resampled_affine',
    'crop_origin',
    'full_resampled_shape',
    'subject_id',
    'preprocessing_version',
    'source_mtt_invalid_voxels',
    'source_mtt_valid_fraction',
}


def as_scalar_text(value):
    return str(np.asarray(value).item())


def file_sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(SOURCE_MANIFEST_PATH)

source_df = pd.read_csv(SOURCE_MANIFEST_PATH)

if len(source_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Expected {EXPECTED_SUBJECTS} source subjects, found {len(source_df)}.'
    )

if source_df['subject_id'].duplicated().any():
    raise RuntimeError('Source manifest contains duplicate subject IDs.')

expected_paths = {
    subject_id: NEW_SUBJECT_DIR / f'{subject_id}_{NEW_VERSION}.npz'
    for subject_id in source_df['subject_id']
}
missing_paths = [
    str(path)
    for path in expected_paths.values()
    if not path.exists()
]

if missing_paths:
    print(f'Missing r1 files: {len(missing_paths)}')
    display(pd.DataFrame({'missing_path': missing_paths}))
    raise RuntimeError(
        'The r1 preprocessing run is incomplete. Rerun its standalone '
        'preprocessing block before verification.'
    )

case_rows = []
channel_rows = []
comparison_rows = []

for case_index, row in enumerate(
    source_df.sort_values('subject_id').itertuples(index=False),
    start=1,
):
    subject_id = row.subject_id
    new_path = expected_paths[subject_id]
    old_path = OLD_SUBJECT_DIR / f'{subject_id}_{OLD_VERSION}.npz'

    errors = []
    case_record = {
        'subject_id': subject_id,
        'path': str(new_path),
        'readable': False,
        'valid': False,
        'image_shape': None,
        'target_binary_voxels': None,
        'target_soft_sum': None,
        'source_mtt_invalid_voxels': None,
        'source_mtt_valid_fraction': None,
        'orientation': None,
        'spacing': None,
        'error': None,
    }

    try:
        with np.load(new_path, allow_pickle=False) as data:
            missing_keys = REQUIRED_KEYS - set(data.files)
            if missing_keys:
                errors.append(f'missing keys: {sorted(missing_keys)}')

            if not missing_keys:
                image = np.asarray(data['image'])
                target_binary = np.asarray(data['target_binary'])
                target_soft = np.asarray(data['target_soft'])
                crop_mask = np.asarray(data['crop_mask'])
                perfusion_valid = np.asarray(data['perfusion_valid'])
                mtt_valid = np.asarray(data['mtt_valid'])
                mtt_support = np.asarray(data['mtt_interpolation_support'])
                cow_label = np.asarray(data['cow_label'])
                cow_soft = np.asarray(data['cow_soft_union'])
                lvo_proximity = np.asarray(data['lvo_proximity'])
                vessel_features = np.asarray(data['vessel_features_raw'])
                affine = np.asarray(data['affine'], dtype=np.float64)

                spatial_shape = tuple(image.shape[1:])
                stored_subject = as_scalar_text(data['subject_id'])
                stored_version = as_scalar_text(data['preprocessing_version'])
                stored_channels = [str(value) for value in data['input_channels'].tolist()]

                if stored_subject != subject_id:
                    errors.append(f'subject mismatch: {stored_subject}')
                if stored_version != NEW_VERSION:
                    errors.append(f'version mismatch: {stored_version}')
                if stored_channels != EXPECTED_INPUT_CHANNELS:
                    errors.append('input channel order mismatch')
                if image.ndim != 4 or image.shape[0] != EXPECTED_CHANNELS:
                    errors.append(f'invalid image shape: {image.shape}')

                shape_arrays = {
                    'target_binary': target_binary,
                    'target_soft': target_soft,
                    'crop_mask': crop_mask,
                    'perfusion_valid': perfusion_valid,
                    'mtt_valid': mtt_valid,
                    'mtt_support': mtt_support,
                    'cow_label': cow_label,
                    'cow_soft': cow_soft,
                    'lvo_proximity': lvo_proximity,
                }

                for name, array in shape_arrays.items():
                    if tuple(array.shape) != spatial_shape:
                        errors.append(
                            f'{name} shape {array.shape} != {spatial_shape}'
                        )

                finite_arrays = {
                    'image': image,
                    'target_soft': target_soft,
                    'mtt_support': mtt_support,
                    'cow_soft': cow_soft,
                    'lvo_proximity': lvo_proximity,
                    'vessel_features': vessel_features,
                    'affine': affine,
                }

                for name, array in finite_arrays.items():
                    if not np.isfinite(array).all():
                        errors.append(f'{name} contains nonfinite values')

                if np.isfinite(image).all():
                    if image.min() < -1.0001 or image.max() > 1.0001:
                        errors.append('image is outside [-1, 1]')

                    non_delta = [
                        index for index in range(EXPECTED_CHANNELS)
                        if index not in {9, 10, 11, 12}
                    ]
                    if image[non_delta].min() < -0.0001:
                        errors.append('non-delta channel is below zero')

                if not set(np.unique(target_binary)).issubset({0, 1}):
                    errors.append('binary target is not binary')
                if not set(np.unique(crop_mask)).issubset({0, 1}):
                    errors.append('crop mask is not binary')
                if not set(np.unique(perfusion_valid)).issubset({0, 1}):
                    errors.append('perfusion-valid mask is not binary')
                if not set(np.unique(mtt_valid)).issubset({0, 1}):
                    errors.append('MTT-valid mask is not binary')
                if target_soft.min() < -0.0001 or target_soft.max() > 1.0001:
                    errors.append('soft target is outside [0, 1]')
                if mtt_support.min() < -0.0001 or mtt_support.max() > 1.0001:
                    errors.append('MTT interpolation support is outside [0, 1]')
                if cow_label.min() < 0 or cow_label.max() > 13:
                    errors.append('CoW labels are outside [0, 13]')
                if vessel_features.shape != (13, 16):
                    errors.append(
                        f'vessel feature shape is {vessel_features.shape}, not (13, 16)'
                    )

                spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
                orientation = ''.join(nib.aff2axcodes(affine))

                if orientation != 'RAS':
                    errors.append(f'orientation is {orientation}, not RAS')
                if not np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0):
                    errors.append(f'spacing is {spacing.tolist()}')

                case_record.update({
                    'readable': True,
                    'image_shape': str(tuple(image.shape)),
                    'target_binary_voxels': int(target_binary.sum()),
                    'target_soft_sum': float(
                        target_soft.sum(dtype=np.float64)
                    ),
                    'source_mtt_invalid_voxels': int(
                        data['source_mtt_invalid_voxels']
                    ),
                    'source_mtt_valid_fraction': float(
                        data['source_mtt_valid_fraction']
                    ),
                    'orientation': orientation,
                    'spacing': str(tuple(float(value) for value in spacing)),
                })

                for channel_index, channel_name in enumerate(EXPECTED_INPUT_CHANNELS):
                    channel = image[channel_index]
                    channel_rows.append({
                        'subject_id': subject_id,
                        'channel_index': channel_index,
                        'channel_name': channel_name,
                        'nonfinite_voxels': int((~np.isfinite(channel)).sum()),
                        'minimum': float(np.nanmin(channel)),
                        'maximum': float(np.nanmax(channel)),
                        'mean': float(np.nanmean(channel, dtype=np.float64)),
                    })

                comparison = {
                    'subject_id': subject_id,
                    'old_file_exists': old_path.exists(),
                    'target_binary_equal': None,
                    'target_soft_equal': None,
                    'cow_label_equal': None,
                    'affine_equal': None,
                    'clean_mtt_case': int(data['source_mtt_invalid_voxels']) == 0,
                    'full_image_equal_for_clean_mtt': None,
                    'error': None,
                }

                if old_path.exists():
                    try:
                        with np.load(old_path, allow_pickle=False) as old:
                            comparison['target_binary_equal'] = bool(
                                np.array_equal(target_binary, old['target_binary'])
                            )
                            comparison['target_soft_equal'] = bool(
                                np.array_equal(target_soft, old['target_soft'])
                            )
                            comparison['cow_label_equal'] = bool(
                                np.array_equal(cow_label, old['cow_label'])
                            )
                            comparison['affine_equal'] = bool(
                                np.allclose(affine, old['affine'], atol=1e-8, rtol=0)
                            )
                            if comparison['clean_mtt_case']:
                                comparison['full_image_equal_for_clean_mtt'] = bool(
                                    np.array_equal(image, old['image'])
                                )
                    except Exception as exc:
                        comparison['error'] = repr(exc)

                comparison_rows.append(comparison)

        case_record['valid'] = len(errors) == 0
        case_record['error'] = ' | '.join(errors) if errors else None

    except Exception as exc:
        case_record['error'] = repr(exc)

    case_rows.append(case_record)

    if case_index == 1 or case_index % 20 == 0 or case_index == EXPECTED_SUBJECTS:
        print(f'Verified {case_index}/{EXPECTED_SUBJECTS}: {subject_id}')

case_df = pd.DataFrame(case_rows)
channel_df = pd.DataFrame(channel_rows)
comparison_df = pd.DataFrame(comparison_rows)

case_df.to_csv(CASE_AUDIT_PATH, index=False)
channel_df.to_csv(CHANNEL_AUDIT_PATH, index=False)
comparison_df.to_csv(COMPARISON_PATH, index=False)

valid_count = int(case_df['valid'].sum())
invalid_df = case_df[~case_df['valid']].copy()

if not NEW_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Missing r1 manifest: {NEW_MANIFEST_PATH}'
    )

new_manifest_df = pd.read_csv(NEW_MANIFEST_PATH)
manifest_split_counts = new_manifest_df['dev_split'].value_counts().to_dict()

geometry_checks = [
    'target_binary_equal',
    'target_soft_equal',
    'cow_label_equal',
    'affine_equal',
]
geometry_ok = True

if len(comparison_df):
    for column in geometry_checks:
        available = comparison_df[column].notna()
        if available.any() and not comparison_df.loc[available, column].all():
            geometry_ok = False

clean_comparison = comparison_df[
    comparison_df['clean_mtt_case']
    & comparison_df['full_image_equal_for_clean_mtt'].notna()
]
clean_images_equal = bool(
    len(clean_comparison) > 0
    and clean_comparison['full_image_equal_for_clean_mtt'].all()
)

gate_passed = bool(
    valid_count == EXPECTED_SUBJECTS
    and len(new_manifest_df) == EXPECTED_SUBJECTS
    and manifest_split_counts == EXPECTED_SPLITS
    and geometry_ok
    and clean_images_equal
)

gate = {
    'preprocessing_version': NEW_VERSION,
    'expected_subjects': EXPECTED_SUBJECTS,
    'valid_subjects': valid_count,
    'manifest_subjects': int(len(new_manifest_df)),
    'manifest_split_counts': manifest_split_counts,
    'geometry_checks_passed': geometry_ok,
    'clean_mtt_images_identical_to_original': clean_images_equal,
    'source_mtt_affected_subjects': int(
        (case_df['source_mtt_invalid_voxels'].fillna(0) > 0).sum()
    ),
    'gate_passed': gate_passed,
    'case_audit_path': str(CASE_AUDIT_PATH),
    'channel_audit_path': str(CHANNEL_AUDIT_PATH),
    'comparison_path': str(COMPARISON_PATH),
    'manifest_sha256': file_sha256(NEW_MANIFEST_PATH),
}

GATE_PATH.write_text(json.dumps(gate, indent=2), encoding='utf-8')

print('\nCase validation counts:')
display(
    case_df['valid'].value_counts(dropna=False)
    .rename_axis('valid')
    .reset_index(name='subjects')
)

print('\nSplit counts:')
display(
    new_manifest_df.groupby('dev_split').size()
    .reset_index(name='subjects')
)

print('\nMTT-invalid source subjects:')
display(
    case_df[case_df['source_mtt_invalid_voxels'].fillna(0) > 0][
        [
            'subject_id',
            'source_mtt_invalid_voxels',
            'source_mtt_valid_fraction',
        ]
    ].sort_values('subject_id')
)

print('\nPer-channel finite/range summary:')
display(
    channel_df.groupby(['channel_index', 'channel_name'], as_index=False)
    .agg(
        cases=('subject_id', 'count'),
        cases_with_nonfinite=(
            'nonfinite_voxels',
            lambda values: int((values > 0).sum()),
        ),
        global_min=('minimum', 'min'),
        global_max=('maximum', 'max'),
        mean_of_case_means=('mean', 'mean'),
    )
)

print('\nOriginal-versus-r1 comparison:')
display(
    pd.DataFrame({
        'check': [
            'Geometry/targets preserved',
            'Clean-MTT images identical',
            'Final gate passed',
        ],
        'passed': [
            geometry_ok,
            clean_images_equal,
            gate_passed,
        ],
    })
)

if len(invalid_df):
    print('\nInvalid r1 cases:')
    display(invalid_df[['subject_id', 'path', 'error']])

if not gate_passed:
    raise RuntimeError(
        'TRIAL8 V2 R1 VERIFICATION FAILED. Do not continue to atlas '
        'processing. Review the saved audits above.'
    )

new_manifest_df.to_csv(FROZEN_MANIFEST_PATH, index=False)

print('\n' + '=' * 80)
print('TRIAL8 V2 R1 VERIFICATION PASSED')
print('=' * 80)
print(f'Valid subjects:          {valid_count}/{EXPECTED_SUBJECTS}')
print(f'Frozen manifest:         {FROZEN_MANIFEST_PATH}')
print(f'Case audit:              {CASE_AUDIT_PATH}')
print(f'Channel audit:           {CHANNEL_AUDIT_PATH}')
print(f'Geometry comparison:     {COMPARISON_PATH}')
print(f'Gate record:             {GATE_PATH}')
print('\nThe dataset is now eligible for vascular-atlas preflight.')


In [ ]:
import gc
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'nibabel',
    'pandas',
])

import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v1_trial0.csv'
)

R1_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r1.csv'
)

R1_SUBJECT_DIR = (
    PROJECT_ROOT
    / 'preprocessing'
    / 'v2_vascufate_ras_0p8x0p8x2_r1'
    / 'subjects_npz'
)

SUSPECT_SUBJECTS = [
    'sub-stroke0077',
    'sub-stroke0109',
    'sub-stroke0137',
    'sub-stroke0138',
    'sub-stroke0152',
]

source_df = pd.read_csv(SOURCE_MANIFEST_PATH)
r1_df = pd.read_csv(R1_MANIFEST_PATH)

assert len(source_df) == 148
assert len(r1_df) == 148

source_df = source_df.set_index('subject_id', drop=False)
r1_df = r1_df.set_index('subject_id', drop=False)


def compact_counts(values, counts):
    return ', '.join(
        f'{float(value):g}:{int(count)}'
        for value, count in zip(values, counts)
    )


records = []

for subject_id in SUSPECT_SUBJECTS:
    if subject_id not in source_df.index:
        raise KeyError(f'{subject_id} is missing from the source manifest.')

    source_path = Path(
        source_df.loc[subject_id, 'path_cow_mask']
    )

    if not source_path.exists():
        raise FileNotFoundError(source_path)

    if 'vascufate_v2_npz_path' in r1_df.columns:
        npz_path = Path(
            r1_df.loc[subject_id, 'vascufate_v2_npz_path']
        )
    else:
        npz_path = (
            R1_SUBJECT_DIR
            / f'{subject_id}_preprocessed_v2_vascufate_ras_0p8x0p8x2_r1.npz'
        )

    if not npz_path.exists():
        matches = sorted(R1_SUBJECT_DIR.glob(f'{subject_id}*.npz'))

        if len(matches) != 1:
            raise FileNotFoundError(
                f'Could not resolve one r1 NPZ for {subject_id}: {matches}'
            )

        npz_path = matches[0]

    source_image = nib.load(str(source_path))
    source_labels = np.asanyarray(source_image.dataobj)

    source_finite = np.isfinite(source_labels)
    source_nonfinite_count = int((~source_finite).sum())

    finite_source_values = source_labels[source_finite]
    source_unique, source_counts = np.unique(
        finite_source_values,
        return_counts=True,
    )

    source_unexpected_mask = (
        (source_unique < 0)
        | (source_unique > 13)
        | (~np.isclose(source_unique, np.rint(source_unique)))
    )

    source_unexpected_values = source_unique[source_unexpected_mask]
    source_unexpected_counts = source_counts[source_unexpected_mask]

    with np.load(npz_path, allow_pickle=False) as data:
        r1_labels = data['cow_label']
        vessel_presence = data['vessel_presence']

        r1_unique, r1_counts = np.unique(
            r1_labels,
            return_counts=True,
        )

        r1_unexpected_mask = (
            (r1_unique < 0)
            | (r1_unique > 13)
        )

        r1_unexpected_values = r1_unique[r1_unexpected_mask]
        r1_unexpected_counts = r1_counts[r1_unexpected_mask]

        records.append({
            'subject_id': subject_id,
            'source_dtype': str(source_labels.dtype),
            'source_min': float(np.nanmin(source_labels)),
            'source_max': float(np.nanmax(source_labels)),
            'source_nonfinite_voxels': source_nonfinite_count,
            'source_unexpected': compact_counts(
                source_unexpected_values,
                source_unexpected_counts,
            ),
            'r1_dtype': str(r1_labels.dtype),
            'r1_min': int(r1_labels.min()),
            'r1_max': int(r1_labels.max()),
            'r1_unexpected': compact_counts(
                r1_unexpected_values,
                r1_unexpected_counts,
            ),
            'r1_value_255_voxels': int((r1_labels == 255).sum()),
            'vessels_present': int(vessel_presence.sum()),
            'source_path': str(source_path),
            'r1_npz_path': str(npz_path),
        })

    del source_labels
    del finite_source_values
    del r1_labels
    gc.collect()

diagnostic_df = pd.DataFrame(records)

print('CoW label diagnostic:')
display(diagnostic_df)

print('\nUnexpected source-label patterns:')
display(
    diagnostic_df[
        [
            'subject_id',
            'source_min',
            'source_max',
            'source_unexpected',
            'r1_max',
            'r1_unexpected',
            'r1_value_255_voxels',
        ]
    ]
)

if (
    (diagnostic_df['r1_max'] == 255).all()
    and diagnostic_df['source_unexpected'].str.contains(
        '-1:',
        regex=False,
    ).all()
):
    print(
        '\nLikely cause confirmed: source label -1 was cast to uint8 '
        'and wrapped to 255. It should be normalized to background 0 '
        'before resampling and storage.'
    )
else:
    print(
        '\nThe pattern is not uniformly explained by -1 to 255 wrapping. '
        'Review the displayed source values before assigning or removing '
        'any labels.'
    )


In [ ]:
import gc
import json
import os
import random
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'nibabel',
    'scipy',
    'openpyxl',
])

import nibabel as nib
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display
from nibabel.processing import resample_from_to, resample_to_output
from scipy import ndimage
from scipy.spatial import cKDTree





drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')

SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v1_trial0.csv'
)

PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
TARGET_SPACING = (0.8, 0.8, 2.0)
SEED = 20240824

PREPROCESSING_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
)

SUBJECT_OUTPUT_DIR = (
    PREPROCESSING_ROOT
    / 'subjects_npz'
)

CONFIG_DIR = PROJECT_ROOT / 'configs'
METADATA_DIR = PROJECT_ROOT / 'metadata'

AUDIT_PATH = (
    PREPROCESSING_ROOT
    / 'preprocessing_audit.csv'
)

PROGRESS_PATH = (
    PREPROCESSING_ROOT
    / 'preprocessing_progress.csv'
)

CONFIG_PATH = (
    CONFIG_DIR
    / f'{PREPROCESSING_VERSION}_config.json'
)

OUTPUT_MANIFEST_PATH = (
    METADATA_DIR
    / 'model_manifest_v2_vascufate_r1.csv'
)

for output_directory in [
    PREPROCESSING_ROOT,
    SUBJECT_OUTPUT_DIR,
    CONFIG_DIR,
    METADATA_DIR,
]:
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

def assert_project_path(path):
    path = Path(path).resolve()
    project_root = PROJECT_ROOT.resolve()

    if (
        path != project_root
        and project_root not in path.parents
    ):
        raise ValueError(
            f'Unsafe path outside project root: {path}'
        )

for output_path in [
    PREPROCESSING_ROOT,
    SUBJECT_OUTPUT_DIR,
    CONFIG_PATH,
    AUDIT_PATH,
    PROGRESS_PATH,
    OUTPUT_MANIFEST_PATH,
]:
    assert_project_path(output_path)

random.seed(SEED)
np.random.seed(SEED)

if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        SOURCE_MANIFEST_PATH
    )

manifest_df = pd.read_csv(
    SOURCE_MANIFEST_PATH
)

if len(manifest_df) != 148:
    raise RuntimeError(
        f'Expected 148 source-manifest subjects; '
        f'found {len(manifest_df)}.'
    )

if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError(
        'Source manifest contains duplicate subject IDs.'
    )

SOURCE_COLUMNS = {
    'ncct': 'path_raw_ncct',
    'cta': 'path_deriv_cta',
    'cbf': 'path_deriv_cbf',
    'cbv': 'path_deriv_cbv',
    'mtt': 'path_deriv_mtt',
    'tmax': 'path_deriv_tmax',
    'cow': 'path_cow_mask',
    'lvo': 'path_lvo_mask',
    'target': 'path_final_infarct_mask',
}

required_columns = [
    'subject_id',
    'dev_split',
    'lesion_volume_bin',
    *SOURCE_COLUMNS.values(),
]

missing_columns = [
    column
    for column in required_columns
    if column not in manifest_df.columns
]

if missing_columns:
    raise ValueError(
        f'Missing required manifest columns: '
        f'{missing_columns}'
    )





INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

STRUCTURAL_CHANNEL_INDICES = [0, 1, 2]
PERFUSION_CHANNEL_INDICES = list(
    range(3, len(INPUT_CHANNELS))
)





MTT_VALID_MIN = 0.0
MTT_VALID_MAX = 120.0
MTT_MIN_INTERPOLATION_SUPPORT = 0.999

VESSEL_NAMES = [
    'BA',
    'R_PCA',
    'L_PCA',
    'R_ICA',
    'R_MCA',
    'L_ICA',
    'L_MCA',
    'R_Pcom',
    'L_Pcom',
    'Acom',
    'R_ACA',
    'L_ACA',
    'third_A2',
]





COW_KNOWN_LABELS = set(range(len(VESSEL_NAMES) + 1))
COW_UNDOCUMENTED_LABELS = {15}

VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

preprocessing_config = {
    'preprocessing_version': PREPROCESSING_VERSION,
    'source_manifest': str(
        SOURCE_MANIFEST_PATH
    ),
    'target_orientation': 'RAS',
    'target_spacing_mm': list(
        TARGET_SPACING
    ),
    'input_channels': INPUT_CHANNELS,
    'structural_channel_indices': (
        STRUCTURAL_CHANNEL_INDICES
    ),
    'perfusion_channel_indices': (
        PERFUSION_CHANNEL_INDICES
    ),
    'image_storage_dtype': 'float16',
    'numeric_revision': 'r1_mtt_sentinel_and_cow_schema_safe',
    'mtt_valid_range_seconds': [
        MTT_VALID_MIN,
        MTT_VALID_MAX,
    ],
    'mtt_min_interpolation_support': (
        MTT_MIN_INTERPOLATION_SUPPORT
    ),
    'continuous_invalid_value_policy': (
        'mask before interpolation; normalized masked interpolation'
    ),
    'contralateral_arithmetic': (
        'masked NumPy operations with no evaluation outside valid pairs'
    ),
    'output_commit_policy': (
        'validate temporary NPZ before atomic os.replace'
    ),
    'target_binary_interpolation': (
        'nearest neighbor'
    ),
    'target_soft_interpolation': (
        'trilinear occupancy'
    ),
    'cow_label_interpolation': (
        'nearest neighbor'
    ),
    'cow_known_labels': sorted(COW_KNOWN_LABELS),
    'cow_undocumented_labels': sorted(COW_UNDOCUMENTED_LABELS),
    'cow_undocumented_label_policy': (
        'preserve in cow_soft_union; map to background in cow_label so no '
        'undocumented anatomical identity is invented'
    ),
    'cow_soft_union_interpolation': (
        'trilinear'
    ),
    'lvo_representation': (
        'native-resolution exponential '
        'distance proximity'
    ),
    'lvo_proximity_scale_mm': 10.0,
    'crop_uses_target': False,
    'crop_margin_mm': [
        16.0,
        16.0,
        10.0,
    ],
    'vessel_names': VESSEL_NAMES,
    'vessel_feature_names': (
        VESSEL_FEATURE_NAMES
    ),
    'seed': SEED,
    'flagged_microlesion': (
        'sub-stroke0171'
    ),
}

CONFIG_PATH.write_text(
    json.dumps(
        preprocessing_config,
        indent=2,
    ),
    encoding='utf-8',
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Manifest subjects: {len(manifest_df)}')
print(f'Preprocessing version: {PREPROCESSING_VERSION}')
print(f'Target spacing: {TARGET_SPACING}')
print(f'Image channels: {len(INPUT_CHANNELS)}')
print(f'Subject output directory: {SUBJECT_OUTPUT_DIR}')





def orientation_code(image):
    return ''.join(
        nib.aff2axcodes(image.affine)
    )

def canonical_image(path):
    image = nib.load(str(path))
    canonical = nib.as_closest_canonical(
        image
    )

    if orientation_code(canonical) != 'RAS':
        raise RuntimeError(
            f'Could not canonicalize to RAS: {path}'
        )

    return canonical

def image_array(image):
    return np.asarray(
        image.dataobj,
        dtype=np.float32,
    )

def voxel_volume_mm3(image):
    return float(
        abs(
            np.linalg.det(
                image.affine[:3, :3]
            )
        )
    )

def grids_match(first, second):
    return (
        tuple(first.shape)
        == tuple(second.shape)
        and np.allclose(
            first.affine,
            second.affine,
            atol=1e-4,
            rtol=1e-5,
        )
    )

def harmonize_image(
    image,
    reference,
    order,
    cval,
):
    if grids_match(image, reference):
        return image, False

    harmonized = resample_from_to(
        image,
        (
            reference.shape,
            reference.affine,
        ),
        order=order,
        mode='constant',
        cval=float(cval),
    )

    if not grids_match(
        harmonized,
        reference,
    ):
        raise RuntimeError(
            'Grid harmonization failed.'
        )

    return harmonized, True

def make_image(
    array,
    affine,
):
    return nib.Nifti1Image(
        np.asarray(
            array,
            dtype=np.float32,
        ),
        affine,
    )

def resample_array(
    array,
    source_affine,
    output_grid,
    order,
    cval=0.0,
):
    result = resample_from_to(
        make_image(
            array,
            source_affine,
        ),
        output_grid,
        order=order,
        mode='constant',
        cval=float(cval),
    )

    return np.asarray(
        result.dataobj,
        dtype=np.float32,
    )

def valid_mtt_mask(array):
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    return (
        np.isfinite(array)
        & (array >= MTT_VALID_MIN)
        & (array <= MTT_VALID_MAX)
    )

def masked_resample_array(
    array,
    valid,
    source_affine,
    output_grid,
    order=1,
    minimum_support=0.999,
    invalid_fill=0.0,
):
    

    array = np.asarray(
        array,
        dtype=np.float32,
    )
    valid = (
        np.asarray(valid, dtype=bool)
        & np.isfinite(array)
    )

    numerator_source = np.zeros_like(
        array,
        dtype=np.float32,
    )
    numerator_source[valid] = array[valid]

    numerator = resample_array(
        numerator_source,
        source_affine,
        output_grid,
        order=order,
        cval=0.0,
    )
    support = resample_array(
        valid.astype(np.float32),
        source_affine,
        output_grid,
        order=order,
        cval=0.0,
    )

    output_valid = (
        np.isfinite(numerator)
        & np.isfinite(support)
        & (support >= float(minimum_support))
    )

    output = np.full(
        numerator.shape,
        float(invalid_fill),
        dtype=np.float32,
    )

    np.divide(
        numerator,
        support,
        out=output,
        where=output_valid,
    )

    output[~output_valid] = float(
        invalid_fill
    )

    if not np.isfinite(output).all():
        raise RuntimeError(
            'Masked resampling produced nonfinite values.'
        )

    return (
        output.astype(np.float32),
        output_valid.astype(bool),
        support.astype(np.float32),
    )





def largest_component_overlapping(
    candidate,
    preferred_mask=None,
):
    candidate = np.asarray(
        candidate,
        dtype=bool,
    )

    labels, component_count = (
        ndimage.label(
            candidate,
            structure=(
                ndimage.generate_binary_structure(
                    3,
                    1,
                )
            ),
        )
    )

    if component_count == 0:
        return np.zeros_like(
            candidate,
            dtype=bool,
        )

    component_sizes = np.bincount(
        labels.ravel()
    )
    component_sizes[0] = 0

    selected_label = None

    if (
        preferred_mask is not None
        and np.asarray(
            preferred_mask,
            dtype=bool,
        ).any()
    ):
        preferred_mask = np.asarray(
            preferred_mask,
            dtype=bool,
        )

        overlap_counts = np.bincount(
            labels[preferred_mask].ravel(),
            minlength=component_count + 1,
        )
        overlap_counts[0] = 0

        if overlap_counts.max() > 0:
            selected_label = int(
                overlap_counts.argmax()
            )

    if selected_label is None:
        selected_label = int(
            component_sizes.argmax()
        )

    return labels == selected_label

def construct_perfusion_validity(
    ncct,
    cbf,
    cbv,
    mtt,
    tmax,
    mtt_valid=None,
):
    finite = (
        np.isfinite(ncct)
        & np.isfinite(cbf)
        & np.isfinite(cbv)
        & np.isfinite(mtt)
        & np.isfinite(tmax)
    )

    if mtt_valid is not None:
        finite &= np.asarray(
            mtt_valid,
            dtype=bool,
        )

    ct_soft_tissue = (
        (ncct > -100.0)
        & (ncct < 200.0)
    )

    perfusion_defined = (
        (tmax > -29.5)
        | (cbf > 0.0)
        | (cbv > 0.0)
        | (mtt > 0.0)
    )

    validity = (
        finite
        & ct_soft_tissue
        & perfusion_defined
    )

    structure = (
        ndimage.generate_binary_structure(
            3,
            1,
        )
    )

    validity = ndimage.binary_closing(
        validity,
        structure=structure,
        iterations=2,
    )

    for z_index in range(
        validity.shape[2]
    ):
        validity[:, :, z_index] = (
            ndimage.binary_fill_holes(
                validity[:, :, z_index]
            )
        )

    validity = (
        largest_component_overlapping(
            validity
        )
    )

    validity = ndimage.binary_dilation(
        validity,
        structure=structure,
        iterations=1,
    )

    if mtt_valid is not None:
        validity &= np.asarray(
            mtt_valid,
            dtype=bool,
        )

    return validity.astype(bool)

def construct_broad_crop_mask(
    ncct,
    perfusion_valid,
    cow_union,
    lvo_proximity,
):
    finite = np.isfinite(ncct)

    head_candidate = (
        finite
        & (ncct > -500.0)
        & (ncct < 3000.0)
    )

    structure = (
        ndimage.generate_binary_structure(
            3,
            1,
        )
    )

    head_candidate = (
        ndimage.binary_closing(
            head_candidate,
            structure=structure,
            iterations=3,
        )
    )

    for z_index in range(
        head_candidate.shape[2]
    ):
        head_candidate[:, :, z_index] = (
            ndimage.binary_fill_holes(
                head_candidate[:, :, z_index]
            )
        )

    head_mask = (
        largest_component_overlapping(
            head_candidate,
            preferred_mask=perfusion_valid,
        )
    )

    head_mask = ndimage.binary_dilation(
        head_mask,
        structure=structure,
        iterations=2,
    )

    crop_mask = (
        head_mask
        | perfusion_valid
        | (cow_union > 0.01)
        | (lvo_proximity > 0.01)
    )

    crop_mask = ndimage.binary_closing(
        crop_mask,
        structure=structure,
        iterations=2,
    )

    return crop_mask.astype(bool)

def bounding_box_with_margin(
    mask,
    spacing,
    margin_mm=(16.0, 16.0, 10.0),
):
    coordinates = np.argwhere(mask)

    if len(coordinates) == 0:
        raise RuntimeError(
            'Crop mask is empty.'
        )

    minimum = coordinates.min(axis=0)
    maximum = coordinates.max(axis=0) + 1

    margin_voxels = np.ceil(
        np.asarray(margin_mm)
        / np.asarray(spacing)
    ).astype(int)

    minimum = np.maximum(
        minimum - margin_voxels,
        0,
    )

    maximum = np.minimum(
        maximum + margin_voxels,
        np.asarray(mask.shape),
    )

    return tuple(
        slice(
            int(minimum[axis]),
            int(maximum[axis]),
        )
        for axis in range(3)
    )

def cropped_affine(
    full_affine,
    crop_slices,
):
    crop_origin = np.asarray([
        crop_slices[0].start,
        crop_slices[1].start,
        crop_slices[2].start,
    ])

    output_affine = full_affine.copy()

    output_affine[:3, 3] = (
        nib.affines.apply_affine(
            full_affine,
            crop_origin,
        )
    )

    return output_affine

def mask_coverage(mask, container):
    mask = np.asarray(mask, dtype=bool)
    container = np.asarray(
        container,
        dtype=bool,
    )

    if not mask.any():
        return np.nan

    return float(
        np.logical_and(
            mask,
            container,
        ).sum()
        / mask.sum()
    )





def scaled_window(
    array,
    minimum,
    maximum,
    valid=None,
):
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    finite = np.isfinite(array)

    if valid is None:
        output_valid = finite
    else:
        output_valid = (
            np.asarray(valid, dtype=bool)
            & finite
        )

    safe_array = np.full_like(
        array,
        float(minimum),
        dtype=np.float32,
    )
    safe_array[output_valid] = (
        array[output_valid]
    )

    output = np.clip(
        safe_array,
        minimum,
        maximum,
    )

    output = (
        output - minimum
    ) / (
        maximum - minimum
    )

    output[~output_valid] = 0.0

    if not np.isfinite(output).all():
        raise RuntimeError(
            'Window scaling produced nonfinite values.'
        )

    return output.astype(np.float32)

def mirror_about_midline(
    array,
    midline_index,
):
    array = np.asarray(
        array,
        dtype=np.float32,
    )

    x_indices = np.arange(
        array.shape[0]
    )

    mirrored_indices = np.rint(
        2.0 * midline_index
        - x_indices
    ).astype(int)

    index_valid = (
        (mirrored_indices >= 0)
        & (
            mirrored_indices
            < array.shape[0]
        )
    )

    output = np.zeros_like(
        array,
        dtype=np.float32,
    )

    output[index_valid] = (
        array[mirrored_indices[index_valid]]
    )

    return output, index_valid

def normalized_delta(
    value,
    mirror_value,
    scale,
    valid_pair,
):
    value = np.asarray(
        value,
        dtype=np.float32,
    )
    mirror_value = np.asarray(
        mirror_value,
        dtype=np.float32,
    )
    valid_pair = (
        np.asarray(valid_pair, dtype=bool)
        & np.isfinite(value)
        & np.isfinite(mirror_value)
    )

    delta = np.zeros_like(
        value,
        dtype=np.float32,
    )

    np.subtract(
        value,
        mirror_value,
        out=delta,
        where=valid_pair,
    )
    delta[valid_pair] /= float(scale)

    delta = np.clip(
        delta,
        -1.0,
        1.0,
    )

    delta[~valid_pair] = 0.0

    if not np.isfinite(delta).all():
        raise RuntimeError(
            'Normalized delta produced nonfinite values.'
        )

    return delta.astype(np.float32)

def relative_ratio(
    value,
    mirror_value,
    denominator_floor,
    valid_pair,
):
    value = np.asarray(
        value,
        dtype=np.float32,
    )
    mirror_value = np.asarray(
        mirror_value,
        dtype=np.float32,
    )

    denominator = (
        mirror_value
        + float(denominator_floor)
    )

    valid_pair = (
        np.asarray(valid_pair, dtype=bool)
        & np.isfinite(value)
        & np.isfinite(denominator)
        & (denominator > 0.0)
    )

    numerator = (
        value
        + float(denominator_floor)
    )

    ratio = np.zeros_like(
        value,
        dtype=np.float32,
    )

    np.divide(
        numerator,
        denominator,
        out=ratio,
        where=valid_pair,
    )

    ratio = np.clip(
        ratio,
        0.0,
        2.0,
    ) / 2.0

    ratio[~valid_pair] = 0.0

    if not np.isfinite(ratio).all():
        raise RuntimeError(
            'Relative ratio produced nonfinite values.'
        )

    return ratio.astype(np.float32)





def world_coordinates(
    voxel_coordinates,
    affine,
):
    return nib.affines.apply_affine(
        affine,
        voxel_coordinates,
    )

def extract_vessel_features(
    cow_labels,
    lvo_mask,
    affine,
    voxel_volume,
    brain_center_world,
    cta,
    cbf,
    cbv,
    mtt,
    tmax,
    perfusion_valid,
):
    feature_matrix = np.zeros(
        (
            len(VESSEL_NAMES),
            len(VESSEL_FEATURE_NAMES),
        ),
        dtype=np.float32,
    )

    presence = np.zeros(
        len(VESSEL_NAMES),
        dtype=np.uint8,
    )

    lvo_coordinates = np.argwhere(
        lvo_mask
    )

    if len(lvo_coordinates):
        lvo_world = world_coordinates(
            lvo_coordinates,
            affine,
        )
        lvo_tree = cKDTree(lvo_world)
        lvo_present = True
    else:
        lvo_tree = None
        lvo_present = False

    zooms = np.sqrt(
        np.sum(
            affine[:3, :3] ** 2,
            axis=0,
        )
    )

    modality_arrays = [
        cta,
        cbf,
        cbv,
        mtt,
        tmax,
    ]

    for vessel_index, vessel_name in enumerate(
        VESSEL_NAMES,
        start=1,
    ):
        vessel_mask = (
            cow_labels == vessel_index
        )

        vessel_coordinates = np.argwhere(
            vessel_mask
        )

        if len(vessel_coordinates) == 0:
            continue

        matrix_index = vessel_index - 1
        presence[matrix_index] = 1

        centroid_voxel = (
            vessel_coordinates.mean(axis=0)
        )

        centroid_world = (
            world_coordinates(
                centroid_voxel,
                affine,
            )
        )

        minimum_voxel = (
            vessel_coordinates.min(axis=0)
        )
        maximum_voxel = (
            vessel_coordinates.max(axis=0)
        )

        extent_mm = (
            maximum_voxel
            - minimum_voxel
            + 1
        ) * zooms

        cta_values = cta[vessel_mask]
        cta_values = cta_values[
            np.isfinite(cta_values)
        ].astype(
            np.float64,
            copy=False,
        )

        if len(cta_values) == 0:
            cta_mean = 0.0
            cta_p90 = 0.0
        else:
            cta_mean = float(
                np.mean(
                    cta_values,
                    dtype=np.float64,
                )
            )
            cta_p90 = float(
                np.percentile(
                    cta_values,
                    90,
                )
            )

        perfusion_means = []

        perfusion_measurement_mask = (
            vessel_mask
            & np.asarray(
                perfusion_valid,
                dtype=bool,
            )
        )

        for modality_array in modality_arrays[1:]:
            values = modality_array[
                perfusion_measurement_mask
            ]
            values = values[
                np.isfinite(values)
            ].astype(
                np.float64,
                copy=False,
            )

            if len(values):
                perfusion_means.append(
                    float(
                        np.mean(
                            values,
                            dtype=np.float64,
                        )
                    )
                )
            else:
                perfusion_means.append(0.0)

        if lvo_present:
            vessel_world = world_coordinates(
                vessel_coordinates,
                affine,
            )

            minimum_lvo_distance = float(
                lvo_tree.query(
                    vessel_world,
                    k=1,
                )[0].min()
            )
        else:
            minimum_lvo_distance = 100.0

        overlap_fraction = float(
            np.logical_and(
                vessel_mask,
                lvo_mask,
            ).sum()
            / vessel_mask.sum()
        )

        feature_matrix[matrix_index] = [
            1.0,
            float(
                vessel_mask.sum()
                * voxel_volume
                / 1000.0
            ),
            float(
                centroid_world[0]
                - brain_center_world[0]
            ),
            float(
                centroid_world[1]
                - brain_center_world[1]
            ),
            float(
                centroid_world[2]
                - brain_center_world[2]
            ),
            float(extent_mm[0]),
            float(extent_mm[1]),
            float(extent_mm[2]),
            cta_mean,
            cta_p90,
            perfusion_means[0],
            perfusion_means[1],
            perfusion_means[2],
            perfusion_means[3],
            minimum_lvo_distance,
            overlap_fraction,
        ]

    return (
        feature_matrix,
        presence,
        int(lvo_present),
    )





REQUIRED_NPZ_KEYS = {
    'image',
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'full_resampled_affine',
    'crop_origin',
    'full_resampled_shape',
    'subject_id',
    'preprocessing_version',
    'source_mtt_invalid_voxels',
    'source_mtt_valid_fraction',
}

def validate_output_npz(path):
    result = {
        'valid': False,
        'error': None,
    }

    try:
        with np.load(
            path,
            allow_pickle=False,
        ) as data:
            missing_keys = (
                REQUIRED_NPZ_KEYS
                - set(data.files)
            )

            if missing_keys:
                raise ValueError(
                    f'Missing NPZ keys: '
                    f'{sorted(missing_keys)}'
                )

            image = data['image']
            target_binary = (
                data['target_binary']
            )
            target_soft = (
                data['target_soft']
            )
            cow_label = data['cow_label']
            crop_mask = data['crop_mask']
            perfusion_valid = (
                data['perfusion_valid']
            )
            mtt_valid = data['mtt_valid']
            mtt_support = (
                data['mtt_interpolation_support']
            )
            cow_soft_union = (
                data['cow_soft_union']
            )
            lvo_proximity = (
                data['lvo_proximity']
            )
            vessel_features = (
                data['vessel_features_raw']
            )

            spatial_shape = tuple(
                image.shape[1:]
            )

            if image.shape[0] != len(
                INPUT_CHANNELS
            ):
                raise ValueError(
                    f'Expected '
                    f'{len(INPUT_CHANNELS)} channels; '
                    f'found {image.shape[0]}.'
                )

            for name, array in [
                (
                    'target_binary',
                    target_binary,
                ),
                ('target_soft', target_soft),
                ('crop_mask', crop_mask),
                (
                    'perfusion_valid',
                    perfusion_valid,
                ),
                ('mtt_valid', mtt_valid),
                (
                    'mtt_interpolation_support',
                    mtt_support,
                ),
                ('cow_label', cow_label),
                (
                    'cow_soft_union',
                    cow_soft_union,
                ),
                (
                    'lvo_proximity',
                    lvo_proximity,
                ),
            ]:
                if tuple(array.shape) != spatial_shape:
                    raise ValueError(
                        f'{name} shape mismatch: '
                        f'{array.shape} vs '
                        f'{spatial_shape}'
                    )

            if not np.isfinite(image).all():
                raise ValueError(
                    'Image contains nonfinite values.'
                )

            if not np.isfinite(
                target_soft
            ).all():
                raise ValueError(
                    'Soft target contains '
                    'nonfinite values.'
                )

            if not np.isfinite(
                lvo_proximity
            ).all():
                raise ValueError(
                    'LVO proximity contains '
                    'nonfinite values.'
                )

            if not np.isfinite(
                vessel_features
            ).all():
                raise ValueError(
                    'Vessel features contain '
                    'nonfinite values.'
                )

            if not np.isfinite(
                mtt_support
            ).all():
                raise ValueError(
                    'MTT interpolation support contains '
                    'nonfinite values.'
                )

            if (
                mtt_support.min() < -0.0001
                or mtt_support.max() > 1.0001
            ):
                raise ValueError(
                    'MTT interpolation support is outside '
                    '[0, 1].'
                )

            if vessel_features.shape != (
                len(VESSEL_NAMES),
                len(VESSEL_FEATURE_NAMES),
            ):
                raise ValueError(
                    'Vessel feature shape is invalid: '
                    f'{vessel_features.shape}.'
                )

            if image.min() < -1.0001:
                raise ValueError(
                    'Image minimum is below -1.'
                )

            if image.max() > 1.0001:
                raise ValueError(
                    'Image maximum is above 1.'
                )

            cow_values = set(
                int(value)
                for value in np.unique(cow_label)
            )

            if not cow_values.issubset(COW_KNOWN_LABELS):
                raise ValueError(
                    'CoW label array contains values outside the documented '
                    f'TopCoW schema 0..{len(VESSEL_NAMES)}: '
                    f'{sorted(cow_values - COW_KNOWN_LABELS)}.'
                )

            non_delta_indices = [
                index
                for index in range(
                    len(INPUT_CHANNELS)
                )
                if index not in {9, 10, 11, 12}
            ]

            if image[non_delta_indices].min() < -0.0001:
                raise ValueError(
                    'A non-delta image channel is below zero.'
                )

            stored_channels = [
                str(value)
                for value in data[
                    'input_channels'
                ].tolist()
            ]

            if stored_channels != INPUT_CHANNELS:
                raise ValueError(
                    'Stored input-channel order does not '
                    'match the configured order.'
                )

            stored_version = str(
                data[
                    'preprocessing_version'
                ].item()
            )

            if stored_version != PREPROCESSING_VERSION:
                raise ValueError(
                    'Stored preprocessing version does '
                    f'not match {PREPROCESSING_VERSION}: '
                    f'{stored_version}.'
                )

            if (
                target_binary.min() < 0
                or target_binary.max() > 1
            ):
                raise ValueError(
                    'Binary target is invalid.'
                )

            result.update({
                'valid': True,
                'image_shape': tuple(
                    int(value)
                    for value in image.shape
                ),
                'target_binary_voxels': int(
                    target_binary.sum()
                ),
                'target_soft_sum': float(
                    target_soft.sum(
                        dtype=np.float64
                    )
                ),
                'lvo_present': int(
                    data['lvo_present']
                ),
                'vessels_present': int(
                    data[
                        'vessel_presence'
                    ].sum()
                ),
                'binary_loss_weight': float(
                    data[
                        'binary_loss_weight'
                    ]
                ),
                'source_mtt_invalid_voxels': int(
                    data[
                        'source_mtt_invalid_voxels'
                    ]
                ),
                'source_mtt_valid_fraction': float(
                    data[
                        'source_mtt_valid_fraction'
                    ]
                ),
            })

    except Exception as exc:
        result['error'] = repr(exc)

    return result





def process_subject(row, output_path):
    subject_id = row['subject_id']

    source_images = {
        role: canonical_image(
            row[column]
        )
        for role, column
        in SOURCE_COLUMNS.items()
    }

    raw_mtt_source = image_array(
        source_images['mtt']
    )
    source_mtt_valid = valid_mtt_mask(
        raw_mtt_source
    )
    source_mtt_invalid_voxels = int(
        (~source_mtt_valid).sum()
    )

    sanitized_mtt_source = np.zeros_like(
        raw_mtt_source,
        dtype=np.float32,
    )
    sanitized_mtt_source[
        source_mtt_valid
    ] = raw_mtt_source[source_mtt_valid]

    mtt_source_image = make_image(
        sanitized_mtt_source,
        source_images['mtt'].affine,
    )
    mtt_valid_source_image = make_image(
        source_mtt_valid.astype(np.float32),
        source_images['mtt'].affine,
    )

    reference_native = (
        source_images['ncct']
    )

    harmonized_images = {
        'ncct': reference_native
    }

    harmonized_roles = []
    native_mtt_valid = None

    for role in [
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'cow',
        'lvo',
        'target',
    ]:
        if role == 'mtt':
            if grids_match(
                mtt_source_image,
                reference_native,
            ):
                harmonized = mtt_source_image
                native_mtt_valid = (
                    image_array(
                        mtt_valid_source_image
                    ) > 0.5
                )
                was_resampled = False
            else:
                (
                    harmonized_mtt_array,
                    native_mtt_valid,
                    _,
                ) = masked_resample_array(
                    array=sanitized_mtt_source,
                    valid=source_mtt_valid,
                    source_affine=(
                        source_images['mtt'].affine
                    ),
                    output_grid=(
                        reference_native.shape,
                        reference_native.affine,
                    ),
                    order=1,
                    minimum_support=(
                        MTT_MIN_INTERPOLATION_SUPPORT
                    ),
                    invalid_fill=0.0,
                )
                harmonized = make_image(
                    harmonized_mtt_array,
                    reference_native.affine,
                )
                was_resampled = True

            harmonized_images[role] = harmonized

            if was_resampled:
                harmonized_roles.append(role)

            continue

        order = (
            1
            if role in {
                'cta',
                'cbf',
                'cbv',
                'mtt',
                'tmax',
            }
            else 0
        )

        cval = {
            'cta': -1000.0,
            'cbf': 0.0,
            'cbv': 0.0,
            'mtt': 0.0,
            'tmax': -30.0,
            'cow': 0.0,
            'lvo': 0.0,
            'target': 0.0,
        }[role]

        harmonized, was_resampled = (
            harmonize_image(
                source_images[role],
                reference_native,
                order=order,
                cval=cval,
            )
        )

        harmonized_images[role] = (
            harmonized
        )

        if was_resampled:
            harmonized_roles.append(role)

    native_arrays = {
        role: image_array(image)
        for role, image
        in harmonized_images.items()
    }

    ncct_native = native_arrays['ncct']
    cta_native = native_arrays['cta']
    cbf_native = native_arrays['cbf']
    cbv_native = native_arrays['cbv']
    mtt_native = native_arrays['mtt']
    tmax_native = native_arrays['tmax']

    cow_native_source = np.rint(
        native_arrays['cow']
    ).astype(np.int16)

    cow_source_values = set(
        int(value)
        for value in np.unique(cow_native_source)
    )
    supported_cow_values = (
        COW_KNOWN_LABELS
        | COW_UNDOCUMENTED_LABELS
    )

    if not cow_source_values.issubset(supported_cow_values):
        raise ValueError(
            f'{subject_id} CoW mask contains unsupported labels: '
            f'{sorted(cow_source_values - supported_cow_values)}.'
        )

    
    
    cow_union_native = cow_native_source > 0
    cow_native = cow_native_source.copy()
    cow_native[
        np.isin(
            cow_native,
            list(COW_UNDOCUMENTED_LABELS),
        )
    ] = 0

    lvo_native = (
        native_arrays['lvo'] > 0.5
    )

    target_native = (
        native_arrays['target'] > 0.5
    )

    target_source_array = (
        image_array(
            source_images['target']
        ) > 0
    )

    source_target_volume_ml = float(
        target_source_array.sum()
        * voxel_volume_mm3(
            source_images['target']
        )
        / 1000.0
    )

    native_perfusion_valid = (
        construct_perfusion_validity(
            ncct=ncct_native,
            cbf=cbf_native,
            cbv=cbv_native,
            mtt=mtt_native,
            tmax=tmax_native,
            mtt_valid=native_mtt_valid,
        )
    )

    native_valid_coordinates = (
        np.argwhere(
            native_perfusion_valid
        )
    )

    if len(native_valid_coordinates):
        native_center_voxel = (
            native_valid_coordinates.min(axis=0)
            + native_valid_coordinates.max(
                axis=0
            )
        ) / 2.0
    else:
        native_center_voxel = (
            np.asarray(
                reference_native.shape
            ) - 1
        ) / 2.0

    brain_center_world = (
        nib.affines.apply_affine(
            reference_native.affine,
            native_center_voxel,
        )
    )

    (
        vessel_features,
        vessel_presence,
        lvo_present,
    ) = extract_vessel_features(
        cow_labels=cow_native,
        lvo_mask=lvo_native,
        affine=reference_native.affine,
        voxel_volume=voxel_volume_mm3(
            reference_native
        ),
        brain_center_world=(
            brain_center_world
        ),
        cta=cta_native,
        cbf=cbf_native,
        cbv=cbv_native,
        mtt=mtt_native,
        tmax=tmax_native,
        perfusion_valid=(
            native_perfusion_valid
        ),
    )

    if lvo_present:
        native_spacing = (
            reference_native.header
            .get_zooms()[:3]
        )

        native_lvo_distance = (
            ndimage.distance_transform_edt(
                ~lvo_native,
                sampling=native_spacing,
            )
        ).astype(np.float32)

        native_lvo_proximity = np.exp(
            -native_lvo_distance / 10.0
        ).astype(np.float32)
    else:
        native_lvo_proximity = (
            np.zeros_like(
                ncct_native,
                dtype=np.float32,
            )
        )

    output_reference = (
        resample_to_output(
            reference_native,
            voxel_sizes=TARGET_SPACING,
            order=1,
            mode='constant',
            cval=-1000.0,
        )
    )

    output_grid = (
        output_reference.shape,
        output_reference.affine,
    )

    ncct = image_array(
        output_reference
    )

    cta = resample_array(
        cta_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=-1000.0,
    )

    cbf = resample_array(
        cbf_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=0.0,
    )

    cbv = resample_array(
        cbv_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=0.0,
    )

    (
        mtt,
        output_mtt_valid,
        output_mtt_support,
    ) = masked_resample_array(
        array=mtt_native,
        valid=native_mtt_valid,
        source_affine=(
            reference_native.affine
        ),
        output_grid=output_grid,
        order=1,
        minimum_support=(
            MTT_MIN_INTERPOLATION_SUPPORT
        ),
        invalid_fill=0.0,
    )

    tmax = resample_array(
        tmax_native,
        reference_native.affine,
        output_grid,
        order=1,
        cval=-30.0,
    )

    target_binary = (
        resample_array(
            target_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        ) > 0.5
    )

    target_soft = np.clip(
        resample_array(
            target_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    cow_label = np.rint(
        resample_array(
            cow_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        )
    ).astype(np.uint8)

    cow_soft_union = np.clip(
        resample_array(
            cow_union_native.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    lvo_proximity = np.clip(
        resample_array(
            native_lvo_proximity,
            reference_native.affine,
            output_grid,
            order=1,
            cval=0.0,
        ),
        0.0,
        1.0,
    )

    perfusion_valid = (
        resample_array(
            native_perfusion_valid.astype(
                np.float32
            ),
            reference_native.affine,
            output_grid,
            order=0,
            cval=0.0,
        ) > 0.5
    )

    perfusion_valid &= output_mtt_valid

    crop_mask_full = (
        construct_broad_crop_mask(
            ncct=ncct,
            perfusion_valid=(
                perfusion_valid
            ),
            cow_union=cow_soft_union,
            lvo_proximity=lvo_proximity,
        )
    )

    crop_slices = (
        bounding_box_with_margin(
            crop_mask_full,
            spacing=TARGET_SPACING,
            margin_mm=(
                16.0,
                16.0,
                10.0,
            ),
        )
    )

    crop_origin = np.asarray([
        crop_slices[0].start,
        crop_slices[1].start,
        crop_slices[2].start,
    ], dtype=np.int32)

    crop_mask = (
        crop_mask_full[crop_slices]
    )

    perfusion_valid_crop = (
        perfusion_valid[crop_slices]
    )

    mtt_valid_crop = (
        output_mtt_valid[crop_slices]
    )

    mtt_support_crop = (
        output_mtt_support[crop_slices]
    )

    target_binary_crop = (
        target_binary[crop_slices]
    )

    target_soft_crop = (
        target_soft[crop_slices]
    )

    cow_label_crop = (
        cow_label[crop_slices]
    )

    cow_soft_union_crop = (
        cow_soft_union[crop_slices]
    )

    lvo_proximity_crop = (
        lvo_proximity[crop_slices]
    )

    
    crop_coordinates = np.argwhere(
        crop_mask_full
    )

    midline_index = float(
        (
            crop_coordinates[:, 0].min()
            + crop_coordinates[:, 0].max()
        ) / 2.0
    )

    mirror_cbf, mirror_index_valid = (
        mirror_about_midline(
            cbf,
            midline_index,
        )
    )

    mirror_cbv, _ = mirror_about_midline(
        cbv,
        midline_index,
    )

    mirror_mtt, _ = mirror_about_midline(
        mtt,
        midline_index,
    )

    mirror_tmax, _ = mirror_about_midline(
        tmax,
        midline_index,
    )

    mirror_valid_float, _ = (
        mirror_about_midline(
            perfusion_valid.astype(
                np.float32
            ),
            midline_index,
        )
    )

    mirror_valid = (
        mirror_valid_float > 0.5
    )

    index_valid_volume = np.broadcast_to(
        mirror_index_valid[
            :,
            None,
            None,
        ],
        perfusion_valid.shape,
    )

    pair_valid = (
        perfusion_valid
        & mirror_valid
        & index_valid_volume
        & output_mtt_valid
        & np.isfinite(cbf)
        & np.isfinite(cbv)
        & np.isfinite(mtt)
        & np.isfinite(tmax)
        & np.isfinite(mirror_cbf)
        & np.isfinite(mirror_cbv)
        & np.isfinite(mirror_mtt)
        & np.isfinite(mirror_tmax)
    )

    relative_cbf = relative_ratio(
        cbf,
        mirror_cbf,
        denominator_floor=5.0,
        valid_pair=pair_valid,
    )

    relative_cbv = relative_ratio(
        cbv,
        mirror_cbv,
        denominator_floor=0.5,
        valid_pair=pair_valid,
    )

    delta_cbf = normalized_delta(
        cbf,
        mirror_cbf,
        scale=100.0,
        valid_pair=pair_valid,
    )

    delta_cbv = normalized_delta(
        cbv,
        mirror_cbv,
        scale=10.0,
        valid_pair=pair_valid,
    )

    delta_mtt = normalized_delta(
        mtt,
        mirror_mtt,
        scale=20.0,
        valid_pair=pair_valid,
    )

    delta_tmax = normalized_delta(
        tmax,
        mirror_tmax,
        scale=15.0,
        valid_pair=pair_valid,
    )

    channel_arrays = [
        scaled_window(
            ncct,
            -20.0,
            100.0,
        )[crop_slices],
        scaled_window(
            ncct,
            -100.0,
            300.0,
        )[crop_slices],
        scaled_window(
            cta,
            -100.0,
            1000.0,
        )[crop_slices],
        scaled_window(
            cbf,
            0.0,
            150.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            cbv,
            0.0,
            10.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            mtt,
            0.0,
            30.0,
            valid=perfusion_valid,
        )[crop_slices],
        scaled_window(
            tmax,
            0.0,
            20.0,
            valid=perfusion_valid,
        )[crop_slices],
        relative_cbf[crop_slices],
        relative_cbv[crop_slices],
        delta_cbf[crop_slices],
        delta_cbv[crop_slices],
        delta_mtt[crop_slices],
        delta_tmax[crop_slices],
        perfusion_valid_crop.astype(
            np.float32
        ),
        pair_valid[crop_slices].astype(
            np.float32
        ),
    ]

    for channel_name, channel_array in zip(
        INPUT_CHANNELS,
        channel_arrays,
    ):
        if not np.isfinite(
            channel_array
        ).all():
            raise RuntimeError(
                f'Channel {channel_name} contains '
                'nonfinite values before storage.'
            )

        if channel_name.startswith('delta_'):
            if (
                channel_array.min() < -1.0001
                or channel_array.max() > 1.0001
            ):
                raise RuntimeError(
                    f'Delta channel {channel_name} is '
                    'outside [-1, 1].'
                )
        elif (
            channel_array.min() < -0.0001
            or channel_array.max() > 1.0001
        ):
            raise RuntimeError(
                f'Channel {channel_name} is outside '
                '[0, 1].'
            )

    image = np.stack(
        channel_arrays,
        axis=0,
    ).astype(np.float16)

    output_affine = cropped_affine(
        output_reference.affine,
        crop_slices,
    )

    output_voxel_volume = (
        voxel_volume_mm3(
            output_reference
        )
    )

    target_binary_volume_ml = float(
        target_binary_crop.sum()
        * output_voxel_volume
        / 1000.0
    )

    target_soft_volume_ml = float(
        target_soft_crop.sum()
        * output_voxel_volume
        / 1000.0
    )

    source_target_nonempty = (
        source_target_volume_ml > 0
    )

    binary_target_nonempty = bool(
        target_binary_crop.any()
    )

    binary_loss_weight = 1.0

    if (
        source_target_nonempty
        and not binary_target_nonempty
    ):
        binary_loss_weight = 0.0

    target_crop_coverage = (
        mask_coverage(
            target_binary,
            crop_mask_full,
        )
    )

    full_resampled_shape = np.asarray(
        output_reference.shape,
        dtype=np.int32,
    )

    temporary_path = Path(
        str(output_path) + '.tmp'
    )

    with temporary_path.open('wb') as file:
        np.savez_compressed(
            file,
            image=image,
            target_binary=(
                target_binary_crop.astype(
                    np.uint8
                )
            ),
            target_soft=(
                target_soft_crop.astype(
                    np.float16
                )
            ),
            binary_loss_weight=np.asarray(
                binary_loss_weight,
                dtype=np.float32,
            ),
            crop_mask=(
                crop_mask.astype(
                    np.uint8
                )
            ),
            perfusion_valid=(
                perfusion_valid_crop.astype(
                    np.uint8
                )
            ),
            mtt_valid=(
                mtt_valid_crop.astype(
                    np.uint8
                )
            ),
            mtt_interpolation_support=(
                np.clip(
                    mtt_support_crop,
                    0.0,
                    1.0,
                ).astype(np.float16)
            ),
            cow_label=(
                cow_label_crop.astype(
                    np.uint8
                )
            ),
            cow_soft_union=(
                cow_soft_union_crop.astype(
                    np.float16
                )
            ),
            lvo_proximity=(
                lvo_proximity_crop.astype(
                    np.float16
                )
            ),
            lvo_present=np.asarray(
                lvo_present,
                dtype=np.uint8,
            ),
            vessel_features_raw=(
                vessel_features.astype(
                    np.float32
                )
            ),
            vessel_presence=(
                vessel_presence.astype(
                    np.uint8
                )
            ),
            input_channels=np.asarray(
                INPUT_CHANNELS
            ),
            structural_channel_indices=(
                np.asarray(
                    STRUCTURAL_CHANNEL_INDICES,
                    dtype=np.int16,
                )
            ),
            perfusion_channel_indices=(
                np.asarray(
                    PERFUSION_CHANNEL_INDICES,
                    dtype=np.int16,
                )
            ),
            vessel_names=np.asarray(
                VESSEL_NAMES
            ),
            vessel_feature_names=np.asarray(
                VESSEL_FEATURE_NAMES
            ),
            affine=output_affine.astype(
                np.float64
            ),
            full_resampled_affine=(
                output_reference.affine.astype(
                    np.float64
                )
            ),
            crop_origin=crop_origin,
            full_resampled_shape=(
                full_resampled_shape
            ),
            target_spacing=np.asarray(
                TARGET_SPACING,
                dtype=np.float32,
            ),
            subject_id=np.asarray(
                subject_id
            ),
            preprocessing_version=np.asarray(
                PREPROCESSING_VERSION
            ),
            source_target_volume_ml=np.asarray(
                source_target_volume_ml,
                dtype=np.float32,
            ),
            target_binary_volume_ml=np.asarray(
                target_binary_volume_ml,
                dtype=np.float32,
            ),
            target_soft_volume_ml=np.asarray(
                target_soft_volume_ml,
                dtype=np.float32,
            ),
            target_crop_coverage=np.asarray(
                target_crop_coverage,
                dtype=np.float32,
            ),
            estimated_midline_full_index=np.asarray(
                midline_index,
                dtype=np.float32,
            ),
            harmonized_roles=np.asarray(
                harmonized_roles
            ),
            source_mtt_invalid_voxels=np.asarray(
                source_mtt_invalid_voxels,
                dtype=np.int64,
            ),
            source_mtt_valid_fraction=np.asarray(
                source_mtt_valid.mean(
                    dtype=np.float64
                ),
                dtype=np.float32,
            ),
        )

    result = validate_output_npz(
        temporary_path
    )

    if not result['valid']:
        temporary_path.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            f'Temporary output validation failed: '
            f'{result["error"]}'
        )

    os.replace(
        temporary_path,
        output_path,
    )

    committed_result = validate_output_npz(
        output_path
    )

    if not committed_result['valid']:
        raise RuntimeError(
            'Committed output failed validation: '
            f'{committed_result["error"]}'
        )

    result = committed_result

    result.update({
        'subject_id': subject_id,
        'output_path': str(output_path),
        'source_target_volume_ml': (
            source_target_volume_ml
        ),
        'target_binary_volume_ml': (
            target_binary_volume_ml
        ),
        'target_soft_volume_ml': (
            target_soft_volume_ml
        ),
        'target_crop_coverage': (
            target_crop_coverage
        ),
        'crop_shape': tuple(
            int(value)
            for value in image.shape[1:]
        ),
        'harmonized_roles': (
            harmonized_roles
        ),
        'source_mtt_invalid_voxels': (
            source_mtt_invalid_voxels
        ),
        'status': 'processed',
    })

    del (
        source_images,
        harmonized_images,
        native_arrays,
        ncct_native,
        cta_native,
        cbf_native,
        cbv_native,
        mtt_native,
        tmax_native,
        cow_native,
        lvo_native,
        target_native,
        target_source_array,
        native_perfusion_valid,
        native_lvo_proximity,
        output_reference,
        ncct,
        cta,
        cbf,
        cbv,
        mtt,
        tmax,
        target_binary,
        target_soft,
        cow_label,
        cow_soft_union,
        lvo_proximity,
        perfusion_valid,
        crop_mask_full,
        mirror_cbf,
        mirror_cbv,
        mirror_mtt,
        mirror_tmax,
        mirror_valid_float,
        mirror_valid,
        pair_valid,
        relative_cbf,
        relative_cbv,
        delta_cbf,
        delta_cbv,
        delta_mtt,
        delta_tmax,
        channel_arrays,
        image,
    )

    gc.collect()

    return result





audit_rows = []

for subject_index, (_, row) in enumerate(
    manifest_df
    .sort_values('subject_id')
    .iterrows(),
    start=1,
):
    subject_id = row['subject_id']

    output_path = (
        SUBJECT_OUTPUT_DIR
        / (
            f'{subject_id}_'
            f'{PREPROCESSING_VERSION}.npz'
        )
    )

    print(
        f'\n{subject_index}/{len(manifest_df)} '
        f'{subject_id}'
    )

    audit_record = {
        'subject_id': subject_id,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': (
            row['lesion_volume_bin']
        ),
        'output_path': str(output_path),
        'status': None,
        'valid': False,
        'error': None,
    }

    try:
        if output_path.exists():
            existing_result = (
                validate_output_npz(
                    output_path
                )
            )

            if existing_result['valid']:
                audit_record.update(
                    existing_result
                )
                audit_record['status'] = (
                    'skipped_valid_existing'
                )

                print(
                    '  Valid existing output found. '
                    'Skipping.'
                )
            else:
                print(
                    '  Existing output is invalid. '
                    'Regenerating.'
                )

                processed_result = (
                    process_subject(
                        row,
                        output_path,
                    )
                )

                audit_record.update(
                    processed_result
                )
        else:
            processed_result = process_subject(
                row,
                output_path,
            )

            audit_record.update(
                processed_result
            )

        print(
            f'  Status: '
            f'{audit_record["status"]}'
        )
        print(
            f'  Image shape: '
            f'{audit_record.get("image_shape")}'
        )
        print(
            f'  Binary target voxels: '
            f'{audit_record.get("target_binary_voxels")}'
        )
        print(
            f'  Vessels present: '
            f'{audit_record.get("vessels_present")}'
        )

    except Exception as exc:
        audit_record.update({
            'status': 'failed',
            'valid': False,
            'error': repr(exc),
        })

        print(f'  FAILED: {exc}')

    audit_rows.append(
        audit_record
    )

    pd.DataFrame(
        audit_rows
    ).to_csv(
        PROGRESS_PATH,
        index=False,
    )

    gc.collect()





final_audit_rows = []

for _, row in (
    manifest_df
    .sort_values('subject_id')
    .iterrows()
):
    subject_id = row['subject_id']

    output_path = (
        SUBJECT_OUTPUT_DIR
        / (
            f'{subject_id}_'
            f'{PREPROCESSING_VERSION}.npz'
        )
    )

    validation = validate_output_npz(
        output_path
    ) if output_path.exists() else {
        'valid': False,
        'error': 'Output file missing.',
    }

    final_audit_rows.append({
        'subject_id': subject_id,
        'dev_split': row['dev_split'],
        'lesion_volume_bin': (
            row['lesion_volume_bin']
        ),
        'output_path': str(output_path),
        **validation,
    })

audit_df = pd.DataFrame(
    final_audit_rows
)

audit_df.to_csv(
    AUDIT_PATH,
    index=False,
)

valid_audit_df = audit_df[
    audit_df['valid']
].copy()

output_manifest_df = (
    manifest_df.merge(
        valid_audit_df[
            [
                'subject_id',
                'output_path',
                'image_shape',
                'target_binary_voxels',
                'target_soft_sum',
                'lvo_present',
                'vessels_present',
                'binary_loss_weight',
                'source_mtt_invalid_voxels',
                'source_mtt_valid_fraction',
            ]
        ],
        on='subject_id',
        how='inner',
        validate='one_to_one',
    )
)

output_manifest_df = (
    output_manifest_df.rename(
        columns={
            'output_path': (
                'vascufate_v2_npz_path'
            ),
            'image_shape': (
                'vascufate_v2_image_shape'
            ),
            'target_binary_voxels': (
                'vascufate_v2_target_binary_voxels'
            ),
            'target_soft_sum': (
                'vascufate_v2_target_soft_sum'
            ),
            'lvo_present': (
                'vascufate_v2_lvo_present'
            ),
            'vessels_present': (
                'vascufate_v2_vessels_present'
            ),
            'binary_loss_weight': (
                'vascufate_v2_binary_loss_weight'
            ),
            'source_mtt_invalid_voxels': (
                'vascufate_v2_source_mtt_invalid_voxels'
            ),
            'source_mtt_valid_fraction': (
                'vascufate_v2_source_mtt_valid_fraction'
            ),
        }
    )
)

expected_split_counts = {
    'train': 102,
    'val': 23,
    'test': 23,
}
observed_split_counts = (
    output_manifest_df[
        'dev_split'
    ].value_counts().to_dict()
)

if observed_split_counts != expected_split_counts:
    raise RuntimeError(
        'Final split-count gate failed. Expected '
        f'{expected_split_counts}, found '
        f'{observed_split_counts}. Do not continue.'
    )

output_manifest_df.to_csv(
    OUTPUT_MANIFEST_PATH,
    index=False,
)

print('\n' + '=' * 80)
print('TRIAL8 V2 CORE PREPROCESSING COMPLETE')
print('=' * 80)

print(f'Total expected subjects: {len(manifest_df)}')
print(f'Valid outputs: {len(valid_audit_df)}')
print(
    f'Failed or missing outputs: '
    f'{len(manifest_df) - len(valid_audit_df)}'
)

print('\nStatus counts:')
display(
    audit_df[
        'valid'
    ].value_counts(
        dropna=False
    ).rename_axis(
        'valid'
    ).reset_index(
        name='subjects'
    )
)

print('\nSplit counts in v2 manifest:')
display(
    output_manifest_df
    .groupby('dev_split')
    .size()
    .reset_index(name='subjects')
)

print('\nBinary loss-weight counts:')
display(
    output_manifest_df[
        'vascufate_v2_binary_loss_weight'
    ].value_counts(
        dropna=False
    ).rename_axis(
        'binary_loss_weight'
    ).reset_index(
        name='subjects'
    )
)

print('\nSubjects with empty binary targets:')
display(
    output_manifest_df[
        output_manifest_df[
            'vascufate_v2_target_binary_voxels'
        ] == 0
    ][
        [
            'subject_id',
            'dev_split',
            'lesion_volume_bin',
            'vascufate_v2_target_soft_sum',
            'vascufate_v2_binary_loss_weight',
        ]
    ]
)

print('\nInvalid outputs:')
display(
    audit_df[
        ~audit_df['valid']
    ][
        [
            'subject_id',
            'output_path',
            'error',
        ]
    ]
)

print(f'\nSaved configuration: {CONFIG_PATH}')
print(f'Saved audit: {AUDIT_PATH}')
print(f'Saved v2 manifest: {OUTPUT_MANIFEST_PATH}')
print(f'Subject outputs: {SUBJECT_OUTPUT_DIR}')

if len(valid_audit_df) != len(
    manifest_df
):
    raise RuntimeError(
        'Final preprocessing gate failed: expected '
        f'{len(manifest_df)} valid subjects but found '
        f'{len(valid_audit_df)}. Rerun this standalone '
        'cell to resume only missing or invalid r1 '
        'subjects. Do not continue to atlas processing.'
    )
else:
    print(
        '\nAll subjects completed and passed '
        'NPZ validation.'
    )

print(
    '\nNo Trial8 model has been trained. '
    'The vascular-territory atlas sidecars '
    'are the next stage.'
)


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'nibabel',
    'pandas',
])

import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v1_trial0.csv'
)

OLD_VERSION = 'v2_vascufate_ras_0p8x0p8x2'
NEW_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
EXPECTED_SUBJECTS = 148
EXPECTED_CHANNELS = 15
EXPECTED_SPACING = np.asarray((0.8, 0.8, 2.0), dtype=np.float64)
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}



MTT_INDEPENDENT_CHANNEL_INDICES = [0, 1, 2, 3, 4, 6]
MTT_INDEPENDENT_CHANNEL_NAMES = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'tmax',
]

COW_KNOWN_LABELS = set(range(14))
COW_UNDOCUMENTED_LABELS = {15}

OLD_SUBJECT_DIR = (
    PROJECT_ROOT / 'preprocessing' / OLD_VERSION / 'subjects_npz'
)
NEW_ROOT = PROJECT_ROOT / 'preprocessing' / NEW_VERSION
NEW_SUBJECT_DIR = NEW_ROOT / 'subjects_npz'
NEW_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1.csv'
)
AUDIT_DIR = NEW_ROOT / 'final_verification'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CASE_AUDIT_PATH = AUDIT_DIR / 'r1_case_audit.csv'
CHANNEL_AUDIT_PATH = AUDIT_DIR / 'r1_channel_audit.csv'
COMPARISON_PATH = AUDIT_DIR / 'r1_vs_original_geometry_audit.csv'
FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
GATE_PATH = AUDIT_DIR / 'r1_verification_gate.json'

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

REQUIRED_KEYS = {
    'image',
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'full_resampled_affine',
    'crop_origin',
    'full_resampled_shape',
    'subject_id',
    'preprocessing_version',
    'source_mtt_invalid_voxels',
    'source_mtt_valid_fraction',
}


def as_scalar_text(value):
    return str(np.asarray(value).item())


def file_sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(SOURCE_MANIFEST_PATH)

source_df = pd.read_csv(SOURCE_MANIFEST_PATH)

if len(source_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Expected {EXPECTED_SUBJECTS} source subjects, found {len(source_df)}.'
    )

if source_df['subject_id'].duplicated().any():
    raise RuntimeError('Source manifest contains duplicate subject IDs.')

expected_paths = {
    subject_id: NEW_SUBJECT_DIR / f'{subject_id}_{NEW_VERSION}.npz'
    for subject_id in source_df['subject_id']
}
missing_paths = [
    str(path)
    for path in expected_paths.values()
    if not path.exists()
]

if missing_paths:
    print(f'Missing r1 files: {len(missing_paths)}')
    display(pd.DataFrame({'missing_path': missing_paths}))
    raise RuntimeError(
        'The r1 preprocessing run is incomplete. Rerun its standalone '
        'preprocessing block before verification.'
    )

case_rows = []
channel_rows = []
comparison_rows = []

for case_index, row in enumerate(
    source_df.sort_values('subject_id').itertuples(index=False),
    start=1,
):
    subject_id = row.subject_id
    new_path = expected_paths[subject_id]
    old_path = OLD_SUBJECT_DIR / f'{subject_id}_{OLD_VERSION}.npz'

    errors = []
    case_record = {
        'subject_id': subject_id,
        'path': str(new_path),
        'readable': False,
        'valid': False,
        'image_shape': None,
        'target_binary_voxels': None,
        'target_soft_sum': None,
        'source_mtt_invalid_voxels': None,
        'source_mtt_valid_fraction': None,
        'orientation': None,
        'spacing': None,
        'error': None,
    }

    try:
        with np.load(new_path, allow_pickle=False) as data:
            missing_keys = REQUIRED_KEYS - set(data.files)
            if missing_keys:
                errors.append(f'missing keys: {sorted(missing_keys)}')

            if not missing_keys:
                image = np.asarray(data['image'])
                target_binary = np.asarray(data['target_binary'])
                target_soft = np.asarray(data['target_soft'])
                crop_mask = np.asarray(data['crop_mask'])
                perfusion_valid = np.asarray(data['perfusion_valid'])
                mtt_valid = np.asarray(data['mtt_valid'])
                mtt_support = np.asarray(data['mtt_interpolation_support'])
                cow_label = np.asarray(data['cow_label'])
                cow_soft = np.asarray(data['cow_soft_union'])
                lvo_proximity = np.asarray(data['lvo_proximity'])
                vessel_features = np.asarray(data['vessel_features_raw'])
                affine = np.asarray(data['affine'], dtype=np.float64)

                spatial_shape = tuple(image.shape[1:])
                stored_subject = as_scalar_text(data['subject_id'])
                stored_version = as_scalar_text(data['preprocessing_version'])
                stored_channels = [str(value) for value in data['input_channels'].tolist()]

                if stored_subject != subject_id:
                    errors.append(f'subject mismatch: {stored_subject}')
                if stored_version != NEW_VERSION:
                    errors.append(f'version mismatch: {stored_version}')
                if stored_channels != EXPECTED_INPUT_CHANNELS:
                    errors.append('input channel order mismatch')
                if image.ndim != 4 or image.shape[0] != EXPECTED_CHANNELS:
                    errors.append(f'invalid image shape: {image.shape}')

                shape_arrays = {
                    'target_binary': target_binary,
                    'target_soft': target_soft,
                    'crop_mask': crop_mask,
                    'perfusion_valid': perfusion_valid,
                    'mtt_valid': mtt_valid,
                    'mtt_support': mtt_support,
                    'cow_label': cow_label,
                    'cow_soft': cow_soft,
                    'lvo_proximity': lvo_proximity,
                }

                for name, array in shape_arrays.items():
                    if tuple(array.shape) != spatial_shape:
                        errors.append(
                            f'{name} shape {array.shape} != {spatial_shape}'
                        )

                finite_arrays = {
                    'image': image,
                    'target_soft': target_soft,
                    'mtt_support': mtt_support,
                    'cow_soft': cow_soft,
                    'lvo_proximity': lvo_proximity,
                    'vessel_features': vessel_features,
                    'affine': affine,
                }

                for name, array in finite_arrays.items():
                    if not np.isfinite(array).all():
                        errors.append(f'{name} contains nonfinite values')

                if np.isfinite(image).all():
                    if image.min() < -1.0001 or image.max() > 1.0001:
                        errors.append('image is outside [-1, 1]')

                    non_delta = [
                        index for index in range(EXPECTED_CHANNELS)
                        if index not in {9, 10, 11, 12}
                    ]
                    if image[non_delta].min() < -0.0001:
                        errors.append('non-delta channel is below zero')

                if not set(np.unique(target_binary)).issubset({0, 1}):
                    errors.append('binary target is not binary')
                if not set(np.unique(crop_mask)).issubset({0, 1}):
                    errors.append('crop mask is not binary')
                if not set(np.unique(perfusion_valid)).issubset({0, 1}):
                    errors.append('perfusion-valid mask is not binary')
                if not set(np.unique(mtt_valid)).issubset({0, 1}):
                    errors.append('MTT-valid mask is not binary')
                if target_soft.min() < -0.0001 or target_soft.max() > 1.0001:
                    errors.append('soft target is outside [0, 1]')
                if mtt_support.min() < -0.0001 or mtt_support.max() > 1.0001:
                    errors.append('MTT interpolation support is outside [0, 1]')
                if cow_label.min() < 0 or cow_label.max() > 13:
                    errors.append('CoW labels are outside [0, 13]')
                if vessel_features.shape != (13, 16):
                    errors.append(
                        f'vessel feature shape is {vessel_features.shape}, not (13, 16)'
                    )

                spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
                orientation = ''.join(nib.aff2axcodes(affine))

                if orientation != 'RAS':
                    errors.append(f'orientation is {orientation}, not RAS')
                if not np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0):
                    errors.append(f'spacing is {spacing.tolist()}')

                case_record.update({
                    'readable': True,
                    'image_shape': str(tuple(image.shape)),
                    'target_binary_voxels': int(target_binary.sum()),
                    'target_soft_sum': float(
                        target_soft.sum(dtype=np.float64)
                    ),
                    'source_mtt_invalid_voxels': int(
                        data['source_mtt_invalid_voxels']
                    ),
                    'source_mtt_valid_fraction': float(
                        data['source_mtt_valid_fraction']
                    ),
                    'orientation': orientation,
                    'spacing': str(tuple(float(value) for value in spacing)),
                })

                for channel_index, channel_name in enumerate(EXPECTED_INPUT_CHANNELS):
                    channel = image[channel_index]
                    channel_rows.append({
                        'subject_id': subject_id,
                        'channel_index': channel_index,
                        'channel_name': channel_name,
                        'nonfinite_voxels': int((~np.isfinite(channel)).sum()),
                        'minimum': float(np.nanmin(channel)),
                        'maximum': float(np.nanmax(channel)),
                        'mean': float(np.nanmean(channel, dtype=np.float64)),
                    })

                comparison = {
                    'subject_id': subject_id,
                    'old_file_exists': old_path.exists(),
                    'target_binary_equal': None,
                    'target_soft_equal': None,
                    'cow_label_equal_after_schema_sanitization': None,
                    'affine_equal': None,
                    'mtt_independent_channels_equal': None,
                    'error': None,
                }

                if old_path.exists():
                    try:
                        with np.load(old_path, allow_pickle=False) as old:
                            old_image = np.asarray(old['image'])
                            old_cow_label = np.asarray(old['cow_label']).copy()

                            old_cow_values = set(
                                int(value)
                                for value in np.unique(old_cow_label)
                            )
                            unsupported_old_cow = (
                                old_cow_values
                                - COW_KNOWN_LABELS
                                - COW_UNDOCUMENTED_LABELS
                            )

                            if unsupported_old_cow:
                                raise ValueError(
                                    'Original v2 CoW labels contain unsupported '
                                    f'values: {sorted(unsupported_old_cow)}'
                                )

                            old_cow_label[
                                np.isin(
                                    old_cow_label,
                                    list(COW_UNDOCUMENTED_LABELS),
                                )
                            ] = 0

                            comparison['target_binary_equal'] = bool(
                                np.array_equal(target_binary, old['target_binary'])
                            )
                            comparison['target_soft_equal'] = bool(
                                np.array_equal(target_soft, old['target_soft'])
                            )
                            comparison[
                                'cow_label_equal_after_schema_sanitization'
                            ] = bool(
                                np.array_equal(cow_label, old_cow_label)
                            )
                            comparison['affine_equal'] = bool(
                                np.allclose(affine, old['affine'], atol=1e-8, rtol=0)
                            )
                            comparison['mtt_independent_channels_equal'] = bool(
                                old_image.shape == image.shape
                                and np.array_equal(
                                    image[MTT_INDEPENDENT_CHANNEL_INDICES],
                                    old_image[MTT_INDEPENDENT_CHANNEL_INDICES],
                                )
                            )
                    except Exception as exc:
                        comparison['error'] = repr(exc)

                comparison_rows.append(comparison)

        case_record['valid'] = len(errors) == 0
        case_record['error'] = ' | '.join(errors) if errors else None

    except Exception as exc:
        case_record['error'] = repr(exc)

    case_rows.append(case_record)

    if case_index == 1 or case_index % 20 == 0 or case_index == EXPECTED_SUBJECTS:
        print(f'Verified {case_index}/{EXPECTED_SUBJECTS}: {subject_id}')

case_df = pd.DataFrame(case_rows)
channel_df = pd.DataFrame(channel_rows)
comparison_df = pd.DataFrame(comparison_rows)

case_df.to_csv(CASE_AUDIT_PATH, index=False)
channel_df.to_csv(CHANNEL_AUDIT_PATH, index=False)
comparison_df.to_csv(COMPARISON_PATH, index=False)

valid_count = int(case_df['valid'].sum())
invalid_df = case_df[~case_df['valid']].copy()

if not NEW_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Missing r1 manifest: {NEW_MANIFEST_PATH}'
    )

new_manifest_df = pd.read_csv(NEW_MANIFEST_PATH)
manifest_split_counts = new_manifest_df['dev_split'].value_counts().to_dict()

geometry_checks = [
    'target_binary_equal',
    'target_soft_equal',
    'cow_label_equal_after_schema_sanitization',
    'affine_equal',
    'mtt_independent_channels_equal',
]
comparison_complete = bool(
    len(comparison_df) == EXPECTED_SUBJECTS
    and comparison_df['old_file_exists'].all()
    and comparison_df['error'].isna().all()
)

geometry_ok = bool(
    comparison_complete
    and all(
        comparison_df[column].fillna(False).all()
        for column in geometry_checks
    )
)

gate_passed = bool(
    valid_count == EXPECTED_SUBJECTS
    and len(new_manifest_df) == EXPECTED_SUBJECTS
    and manifest_split_counts == EXPECTED_SPLITS
    and geometry_ok
)

gate = {
    'preprocessing_version': NEW_VERSION,
    'expected_subjects': EXPECTED_SUBJECTS,
    'valid_subjects': valid_count,
    'manifest_subjects': int(len(new_manifest_df)),
    'manifest_split_counts': manifest_split_counts,
    'comparison_complete': comparison_complete,
    'geometry_checks_passed': geometry_ok,
    'mtt_independent_channels': MTT_INDEPENDENT_CHANNEL_NAMES,
    'mtt_independent_channels_preserved': bool(
        len(comparison_df) == EXPECTED_SUBJECTS
        and comparison_df[
            'mtt_independent_channels_equal'
        ].fillna(False).all()
    ),
    'source_mtt_outside_configured_range_subjects': int(
        (case_df['source_mtt_invalid_voxels'].fillna(0) > 0).sum()
    ),
    'gate_passed': gate_passed,
    'case_audit_path': str(CASE_AUDIT_PATH),
    'channel_audit_path': str(CHANNEL_AUDIT_PATH),
    'comparison_path': str(COMPARISON_PATH),
    'manifest_sha256': file_sha256(NEW_MANIFEST_PATH),
}

GATE_PATH.write_text(json.dumps(gate, indent=2), encoding='utf-8')

print('\nCase validation counts:')
display(
    case_df['valid'].value_counts(dropna=False)
    .rename_axis('valid')
    .reset_index(name='subjects')
)

print('\nSplit counts:')
display(
    new_manifest_df.groupby('dev_split').size()
    .reset_index(name='subjects')
)

print('\nSource MTT voxels outside the configured physical range:')
print(
    'This includes expected background/edge voxels and is informational. '
    'It is not a case-level failure.'
)
display(
    case_df[case_df['source_mtt_invalid_voxels'].fillna(0) > 0][
        [
            'subject_id',
            'source_mtt_invalid_voxels',
            'source_mtt_valid_fraction',
        ]
    ].sort_values('subject_id')
)

print('\nPer-channel finite/range summary:')
display(
    channel_df.groupby(['channel_index', 'channel_name'], as_index=False)
    .agg(
        cases=('subject_id', 'count'),
        cases_with_nonfinite=(
            'nonfinite_voxels',
            lambda values: int((values > 0).sum()),
        ),
        global_min=('minimum', 'min'),
        global_max=('maximum', 'max'),
        mean_of_case_means=('mean', 'mean'),
    )
)

print('\nOriginal-versus-r1 comparison:')
display(
    pd.DataFrame({
        'check': [
            'Comparison complete',
            'Geometry/targets/schema-safe CoW preserved',
            'MTT-independent image channels preserved',
            'Final gate passed',
        ],
        'passed': [
            comparison_complete,
            geometry_ok,
            bool(
                len(comparison_df) == EXPECTED_SUBJECTS
                and comparison_df[
                    'mtt_independent_channels_equal'
                ].fillna(False).all()
            ),
            gate_passed,
        ],
    })
)

if len(invalid_df):
    print('\nInvalid r1 cases:')
    display(invalid_df[['subject_id', 'path', 'error']])

if not gate_passed:
    raise RuntimeError(
        'TRIAL8 V2 R1 VERIFICATION FAILED. Do not continue to atlas '
        'processing. Review the saved audits above.'
    )

frozen_manifest_df = new_manifest_df.copy()
case_metrics = case_df.set_index('subject_id')

frozen_manifest_df['vascufate_v2_target_binary_voxels'] = (
    frozen_manifest_df['subject_id'].map(
        case_metrics['target_binary_voxels']
    ).astype(np.int64)
)
frozen_manifest_df['vascufate_v2_target_soft_sum'] = (
    frozen_manifest_df['subject_id'].map(
        case_metrics['target_soft_sum']
    ).astype(np.float64)
)

if not np.isfinite(
    frozen_manifest_df['vascufate_v2_target_soft_sum']
).all():
    raise RuntimeError(
        'Refusing to freeze manifest because target-soft sums are nonfinite.'
    )

frozen_manifest_df.to_csv(FROZEN_MANIFEST_PATH, index=False)

print('\n' + '=' * 80)
print('TRIAL8 V2 R1 VERIFICATION PASSED')
print('=' * 80)
print(f'Valid subjects:          {valid_count}/{EXPECTED_SUBJECTS}')
print(f'Frozen manifest:         {FROZEN_MANIFEST_PATH}')
print(f'Case audit:              {CASE_AUDIT_PATH}')
print(f'Channel audit:           {CHANNEL_AUDIT_PATH}')
print(f'Geometry comparison:     {COMPARISON_PATH}')
print(f'Gate record:             {GATE_PATH}')
print('\nThe dataset is now eligible for vascular-atlas preflight.')


In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v1_trial0.csv'
)

OLD_VERSION = 'v2_vascufate_ras_0p8x0p8x2'
NEW_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'

OLD_SUBJECT_DIR = (
    PROJECT_ROOT / 'preprocessing' / OLD_VERSION / 'subjects_npz'
)
NEW_SUBJECT_DIR = (
    PROJECT_ROOT / 'preprocessing' / NEW_VERSION / 'subjects_npz'
)
AUDIT_DIR = (
    PROJECT_ROOT
    / 'preprocessing'
    / NEW_VERSION
    / 'final_verification'
)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CASE_OUTPUT_PATH = AUDIT_DIR / 'r1_equivalence_diagnostic_cases.csv'
CHANNEL_OUTPUT_PATH = AUDIT_DIR / 'r1_equivalence_diagnostic_channels.csv'

CHANNEL_NAMES = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]


STABLE_CHANNEL_INDICES = [0, 1, 2, 3, 4, 6]

manifest_df = pd.read_csv(SOURCE_MANIFEST_PATH)

if len(manifest_df) != 148:
    raise RuntimeError(
        f'Expected 148 subjects; found {len(manifest_df)}.'
    )


def array_difference_summary(new_array, old_array):
    new_array = np.asarray(new_array)
    old_array = np.asarray(old_array)

    if new_array.shape != old_array.shape:
        return {
            'shape_equal': False,
            'exact_equal': False,
            'nonfinite_old': int((~np.isfinite(old_array)).sum()),
            'nonfinite_new': int((~np.isfinite(new_array)).sum()),
            'changed_voxels': -1,
            'changed_gt_5e_4': -1,
            'changed_gt_1e_3': -1,
            'max_abs_difference': np.inf,
            'mean_abs_difference': np.inf,
        }

    old_float = old_array.astype(np.float32, copy=False)
    new_float = new_array.astype(np.float32, copy=False)

    finite_pair = np.isfinite(old_float) & np.isfinite(new_float)
    nonfinite_old = int((~np.isfinite(old_float)).sum())
    nonfinite_new = int((~np.isfinite(new_float)).sum())

    exact_equal = bool(np.array_equal(new_array, old_array))
    changed_voxels = int(np.count_nonzero(new_array != old_array))

    if finite_pair.any():
        absolute_difference = np.abs(
            new_float[finite_pair] - old_float[finite_pair]
        )
        max_abs_difference = float(absolute_difference.max())
        mean_abs_difference = float(
            absolute_difference.mean(dtype=np.float64)
        )
        changed_gt_5e_4 = int((absolute_difference > 5e-4).sum())
        changed_gt_1e_3 = int((absolute_difference > 1e-3).sum())
    else:
        max_abs_difference = np.inf
        mean_abs_difference = np.inf
        changed_gt_5e_4 = -1
        changed_gt_1e_3 = -1

    return {
        'shape_equal': True,
        'exact_equal': exact_equal,
        'nonfinite_old': nonfinite_old,
        'nonfinite_new': nonfinite_new,
        'changed_voxels': changed_voxels,
        'changed_gt_5e_4': changed_gt_5e_4,
        'changed_gt_1e_3': changed_gt_1e_3,
        'max_abs_difference': max_abs_difference,
        'mean_abs_difference': mean_abs_difference,
    }


case_rows = []
channel_rows = []

for case_index, subject_id in enumerate(
    sorted(manifest_df['subject_id'].tolist()),
    start=1,
):
    old_path = OLD_SUBJECT_DIR / f'{subject_id}_{OLD_VERSION}.npz'
    new_path = NEW_SUBJECT_DIR / f'{subject_id}_{NEW_VERSION}.npz'

    if not old_path.exists():
        raise FileNotFoundError(old_path)
    if not new_path.exists():
        raise FileNotFoundError(new_path)

    with np.load(old_path, allow_pickle=False) as old_data:
        with np.load(new_path, allow_pickle=False) as new_data:
            old_image = np.asarray(old_data['image'])
            new_image = np.asarray(new_data['image'])

            old_target_binary = np.asarray(old_data['target_binary'])
            new_target_binary = np.asarray(new_data['target_binary'])
            old_target_soft = np.asarray(old_data['target_soft'])
            new_target_soft = np.asarray(new_data['target_soft'])

            old_cow = np.asarray(old_data['cow_label']).copy()
            new_cow = np.asarray(new_data['cow_label'])
            old_cow[old_cow == 15] = 0

            old_affine = np.asarray(old_data['affine'], dtype=np.float64)
            new_affine = np.asarray(new_data['affine'], dtype=np.float64)

            binary_summary = array_difference_summary(
                new_target_binary,
                old_target_binary,
            )
            soft_summary = array_difference_summary(
                new_target_soft,
                old_target_soft,
            )
            cow_summary = array_difference_summary(new_cow, old_cow)
            affine_summary = array_difference_summary(new_affine, old_affine)

            stable_changed_total = 0
            stable_changed_gt_5e_4_total = 0
            stable_changed_gt_1e_3_total = 0
            stable_max_abs_difference = 0.0
            stable_all_exact = True

            for channel_index in STABLE_CHANNEL_INDICES:
                channel_summary = array_difference_summary(
                    new_image[channel_index],
                    old_image[channel_index],
                )

                channel_rows.append({
                    'subject_id': subject_id,
                    'channel_index': channel_index,
                    'channel_name': CHANNEL_NAMES[channel_index],
                    **channel_summary,
                })

                stable_changed_total += channel_summary['changed_voxels']
                stable_changed_gt_5e_4_total += (
                    channel_summary['changed_gt_5e_4']
                )
                stable_changed_gt_1e_3_total += (
                    channel_summary['changed_gt_1e_3']
                )
                stable_max_abs_difference = max(
                    stable_max_abs_difference,
                    channel_summary['max_abs_difference'],
                )
                stable_all_exact &= channel_summary['exact_equal']

            case_rows.append({
                'subject_id': subject_id,
                'target_binary_exact': binary_summary['exact_equal'],
                'target_binary_changed_voxels': (
                    binary_summary['changed_voxels']
                ),
                'target_soft_exact': soft_summary['exact_equal'],
                'target_soft_changed_voxels': soft_summary['changed_voxels'],
                'target_soft_max_abs_difference': (
                    soft_summary['max_abs_difference']
                ),
                'cow_exact_after_15_to_0': cow_summary['exact_equal'],
                'cow_changed_voxels_after_15_to_0': (
                    cow_summary['changed_voxels']
                ),
                'affine_exact': affine_summary['exact_equal'],
                'affine_max_abs_difference': (
                    affine_summary['max_abs_difference']
                ),
                'stable_channels_all_exact': stable_all_exact,
                'stable_changed_voxels_total': stable_changed_total,
                'stable_changed_gt_5e_4_total': (
                    stable_changed_gt_5e_4_total
                ),
                'stable_changed_gt_1e_3_total': (
                    stable_changed_gt_1e_3_total
                ),
                'stable_max_abs_difference': stable_max_abs_difference,
            })

    del old_image, new_image
    gc.collect()

    if case_index == 1 or case_index % 20 == 0 or case_index == 148:
        print(f'Compared {case_index}/148: {subject_id}')

case_df = pd.DataFrame(case_rows)
channel_df = pd.DataFrame(channel_rows)

case_df.to_csv(CASE_OUTPUT_PATH, index=False)
channel_df.to_csv(CHANNEL_OUTPUT_PATH, index=False)

print('\nExact comparison counts:')
display(
    pd.DataFrame({
        'check': [
            'Binary targets exact',
            'Soft targets exact',
            'CoW exact after old label 15 -> 0',
            'Affines exact',
            'Stable image channels exact',
        ],
        'passing_subjects': [
            int(case_df['target_binary_exact'].sum()),
            int(case_df['target_soft_exact'].sum()),
            int(case_df['cow_exact_after_15_to_0'].sum()),
            int(case_df['affine_exact'].sum()),
            int(case_df['stable_channels_all_exact'].sum()),
        ],
        'total_subjects': [148] * 5,
    })
)

print('\nStable-channel difference summary:')
channel_summary_df = (
    channel_df
    .groupby(['channel_index', 'channel_name'], as_index=False)
    .agg(
        cases=('subject_id', 'count'),
        cases_not_exact=('exact_equal', lambda values: int((~values).sum())),
        total_changed_voxels=('changed_voxels', 'sum'),
        total_changed_gt_5e_4=('changed_gt_5e_4', 'sum'),
        total_changed_gt_1e_3=('changed_gt_1e_3', 'sum'),
        maximum_abs_difference=('max_abs_difference', 'max'),
        mean_abs_difference=('mean_abs_difference', 'mean'),
        old_nonfinite_voxels=('nonfinite_old', 'sum'),
        new_nonfinite_voxels=('nonfinite_new', 'sum'),
    )
)
display(channel_summary_df)

print('\nSubjects with any target, CoW, or affine difference:')
structural_failure_df = case_df[
    (~case_df['target_binary_exact'])
    | (~case_df['target_soft_exact'])
    | (~case_df['cow_exact_after_15_to_0'])
    | (~case_df['affine_exact'])
]
display(structural_failure_df)

print('\nSubjects with the largest stable-channel differences:')
display(
    case_df.sort_values(
        [
            'stable_max_abs_difference',
            'stable_changed_voxels_total',
        ],
        ascending=False,
    ).head(20)
)

print(f'\nSaved case diagnostic: {CASE_OUTPUT_PATH}')
print(f'Saved channel diagnostic: {CHANNEL_OUTPUT_PATH}')
print('\nThis cell is read-only. Do not change the verification gate yet.')


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'nibabel',
    'pandas',
])

import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v1_trial0.csv'
)

OLD_VERSION = 'v2_vascufate_ras_0p8x0p8x2'
NEW_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
EXPECTED_SUBJECTS = 148
EXPECTED_CHANNELS = 15
EXPECTED_SPACING = np.asarray((0.8, 0.8, 2.0), dtype=np.float64)
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}



STRUCTURAL_CHANNEL_INDICES = [0, 1, 2]
STRUCTURAL_CHANNEL_NAMES = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
]




SHARED_VALID_PERFUSION_CHANNEL_INDICES = [3, 4, 6]
SHARED_VALID_PERFUSION_CHANNEL_NAMES = [
    'cbf',
    'cbv',
    'tmax',
]

COW_KNOWN_LABELS = set(range(14))
COW_UNDOCUMENTED_LABELS = {15}

OLD_SUBJECT_DIR = (
    PROJECT_ROOT / 'preprocessing' / OLD_VERSION / 'subjects_npz'
)
NEW_ROOT = PROJECT_ROOT / 'preprocessing' / NEW_VERSION
NEW_SUBJECT_DIR = NEW_ROOT / 'subjects_npz'
NEW_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1.csv'
)
AUDIT_DIR = NEW_ROOT / 'final_verification'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CASE_AUDIT_PATH = AUDIT_DIR / 'r1_case_audit.csv'
CHANNEL_AUDIT_PATH = AUDIT_DIR / 'r1_channel_audit.csv'
COMPARISON_PATH = AUDIT_DIR / 'r1_vs_original_geometry_audit.csv'
FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
GATE_PATH = AUDIT_DIR / 'r1_verification_gate.json'

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

REQUIRED_KEYS = {
    'image',
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'full_resampled_affine',
    'crop_origin',
    'full_resampled_shape',
    'subject_id',
    'preprocessing_version',
    'source_mtt_invalid_voxels',
    'source_mtt_valid_fraction',
}


def as_scalar_text(value):
    return str(np.asarray(value).item())


def file_sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


if not SOURCE_MANIFEST_PATH.exists():
    raise FileNotFoundError(SOURCE_MANIFEST_PATH)

source_df = pd.read_csv(SOURCE_MANIFEST_PATH)

if len(source_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Expected {EXPECTED_SUBJECTS} source subjects, found {len(source_df)}.'
    )

if source_df['subject_id'].duplicated().any():
    raise RuntimeError('Source manifest contains duplicate subject IDs.')

expected_paths = {
    subject_id: NEW_SUBJECT_DIR / f'{subject_id}_{NEW_VERSION}.npz'
    for subject_id in source_df['subject_id']
}
missing_paths = [
    str(path)
    for path in expected_paths.values()
    if not path.exists()
]

if missing_paths:
    print(f'Missing r1 files: {len(missing_paths)}')
    display(pd.DataFrame({'missing_path': missing_paths}))
    raise RuntimeError(
        'The r1 preprocessing run is incomplete. Rerun its standalone '
        'preprocessing block before verification.'
    )

case_rows = []
channel_rows = []
comparison_rows = []

for case_index, row in enumerate(
    source_df.sort_values('subject_id').itertuples(index=False),
    start=1,
):
    subject_id = row.subject_id
    new_path = expected_paths[subject_id]
    old_path = OLD_SUBJECT_DIR / f'{subject_id}_{OLD_VERSION}.npz'

    errors = []
    case_record = {
        'subject_id': subject_id,
        'path': str(new_path),
        'readable': False,
        'valid': False,
        'image_shape': None,
        'target_binary_voxels': None,
        'target_soft_sum': None,
        'source_mtt_invalid_voxels': None,
        'source_mtt_valid_fraction': None,
        'orientation': None,
        'spacing': None,
        'error': None,
    }

    try:
        with np.load(new_path, allow_pickle=False) as data:
            missing_keys = REQUIRED_KEYS - set(data.files)
            if missing_keys:
                errors.append(f'missing keys: {sorted(missing_keys)}')

            if not missing_keys:
                image = np.asarray(data['image'])
                target_binary = np.asarray(data['target_binary'])
                target_soft = np.asarray(data['target_soft'])
                crop_mask = np.asarray(data['crop_mask'])
                perfusion_valid = np.asarray(data['perfusion_valid'])
                mtt_valid = np.asarray(data['mtt_valid'])
                mtt_support = np.asarray(data['mtt_interpolation_support'])
                cow_label = np.asarray(data['cow_label'])
                cow_soft = np.asarray(data['cow_soft_union'])
                lvo_proximity = np.asarray(data['lvo_proximity'])
                vessel_features = np.asarray(data['vessel_features_raw'])
                affine = np.asarray(data['affine'], dtype=np.float64)

                spatial_shape = tuple(image.shape[1:])
                stored_subject = as_scalar_text(data['subject_id'])
                stored_version = as_scalar_text(data['preprocessing_version'])
                stored_channels = [str(value) for value in data['input_channels'].tolist()]

                if stored_subject != subject_id:
                    errors.append(f'subject mismatch: {stored_subject}')
                if stored_version != NEW_VERSION:
                    errors.append(f'version mismatch: {stored_version}')
                if stored_channels != EXPECTED_INPUT_CHANNELS:
                    errors.append('input channel order mismatch')
                if image.ndim != 4 or image.shape[0] != EXPECTED_CHANNELS:
                    errors.append(f'invalid image shape: {image.shape}')

                shape_arrays = {
                    'target_binary': target_binary,
                    'target_soft': target_soft,
                    'crop_mask': crop_mask,
                    'perfusion_valid': perfusion_valid,
                    'mtt_valid': mtt_valid,
                    'mtt_support': mtt_support,
                    'cow_label': cow_label,
                    'cow_soft': cow_soft,
                    'lvo_proximity': lvo_proximity,
                }

                for name, array in shape_arrays.items():
                    if tuple(array.shape) != spatial_shape:
                        errors.append(
                            f'{name} shape {array.shape} != {spatial_shape}'
                        )

                finite_arrays = {
                    'image': image,
                    'target_soft': target_soft,
                    'mtt_support': mtt_support,
                    'cow_soft': cow_soft,
                    'lvo_proximity': lvo_proximity,
                    'vessel_features': vessel_features,
                    'affine': affine,
                }

                for name, array in finite_arrays.items():
                    if not np.isfinite(array).all():
                        errors.append(f'{name} contains nonfinite values')

                if np.isfinite(image).all():
                    if image.min() < -1.0001 or image.max() > 1.0001:
                        errors.append('image is outside [-1, 1]')

                    non_delta = [
                        index for index in range(EXPECTED_CHANNELS)
                        if index not in {9, 10, 11, 12}
                    ]
                    if image[non_delta].min() < -0.0001:
                        errors.append('non-delta channel is below zero')

                if not set(np.unique(target_binary)).issubset({0, 1}):
                    errors.append('binary target is not binary')
                if not set(np.unique(crop_mask)).issubset({0, 1}):
                    errors.append('crop mask is not binary')
                if not set(np.unique(perfusion_valid)).issubset({0, 1}):
                    errors.append('perfusion-valid mask is not binary')
                if not set(np.unique(mtt_valid)).issubset({0, 1}):
                    errors.append('MTT-valid mask is not binary')
                if target_soft.min() < -0.0001 or target_soft.max() > 1.0001:
                    errors.append('soft target is outside [0, 1]')
                if mtt_support.min() < -0.0001 or mtt_support.max() > 1.0001:
                    errors.append('MTT interpolation support is outside [0, 1]')
                if cow_label.min() < 0 or cow_label.max() > 13:
                    errors.append('CoW labels are outside [0, 13]')
                if vessel_features.shape != (13, 16):
                    errors.append(
                        f'vessel feature shape is {vessel_features.shape}, not (13, 16)'
                    )

                spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
                orientation = ''.join(nib.aff2axcodes(affine))

                if orientation != 'RAS':
                    errors.append(f'orientation is {orientation}, not RAS')
                if not np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0):
                    errors.append(f'spacing is {spacing.tolist()}')

                case_record.update({
                    'readable': True,
                    'image_shape': str(tuple(image.shape)),
                    'target_binary_voxels': int(target_binary.sum()),
                    'target_soft_sum': float(
                        target_soft.sum(dtype=np.float64)
                    ),
                    'source_mtt_invalid_voxels': int(
                        data['source_mtt_invalid_voxels']
                    ),
                    'source_mtt_valid_fraction': float(
                        data['source_mtt_valid_fraction']
                    ),
                    'orientation': orientation,
                    'spacing': str(tuple(float(value) for value in spacing)),
                })

                for channel_index, channel_name in enumerate(EXPECTED_INPUT_CHANNELS):
                    channel = image[channel_index]
                    channel_rows.append({
                        'subject_id': subject_id,
                        'channel_index': channel_index,
                        'channel_name': channel_name,
                        'nonfinite_voxels': int((~np.isfinite(channel)).sum()),
                        'minimum': float(np.nanmin(channel)),
                        'maximum': float(np.nanmax(channel)),
                        'mean': float(np.nanmean(channel, dtype=np.float64)),
                    })

                comparison = {
                    'subject_id': subject_id,
                    'old_file_exists': old_path.exists(),
                    'target_binary_equal': None,
                    'target_soft_equal': None,
                    'cow_label_equal_after_schema_sanitization': None,
                    'affine_equal': None,
                    'structural_channels_equal': None,
                    'shared_valid_perfusion_channels_equal': None,
                    'shared_valid_voxels': None,
                    'old_valid_new_invalid_voxels': None,
                    'new_valid_old_invalid_voxels': None,
                    'error': None,
                }

                if old_path.exists():
                    try:
                        with np.load(old_path, allow_pickle=False) as old:
                            old_image = np.asarray(old['image'])
                            old_cow_label = np.asarray(old['cow_label']).copy()
                            old_perfusion_valid = np.asarray(
                                old['perfusion_valid']
                            ).astype(bool)
                            new_perfusion_valid = perfusion_valid.astype(bool)

                            old_cow_values = set(
                                int(value)
                                for value in np.unique(old_cow_label)
                            )
                            unsupported_old_cow = (
                                old_cow_values
                                - COW_KNOWN_LABELS
                                - COW_UNDOCUMENTED_LABELS
                            )

                            if unsupported_old_cow:
                                raise ValueError(
                                    'Original v2 CoW labels contain unsupported '
                                    f'values: {sorted(unsupported_old_cow)}'
                                )

                            old_cow_label[
                                np.isin(
                                    old_cow_label,
                                    list(COW_UNDOCUMENTED_LABELS),
                                )
                            ] = 0

                            comparison['target_binary_equal'] = bool(
                                np.array_equal(target_binary, old['target_binary'])
                            )
                            comparison['target_soft_equal'] = bool(
                                np.array_equal(target_soft, old['target_soft'])
                            )
                            comparison[
                                'cow_label_equal_after_schema_sanitization'
                            ] = bool(
                                np.array_equal(cow_label, old_cow_label)
                            )
                            comparison['affine_equal'] = bool(
                                np.allclose(affine, old['affine'], atol=1e-8, rtol=0)
                            )
                            comparison['structural_channels_equal'] = bool(
                                old_image.shape == image.shape
                                and np.array_equal(
                                    image[STRUCTURAL_CHANNEL_INDICES],
                                    old_image[STRUCTURAL_CHANNEL_INDICES],
                                )
                            )

                            if old_perfusion_valid.shape != new_perfusion_valid.shape:
                                raise ValueError(
                                    'Old and new perfusion-valid masks have '
                                    'different shapes.'
                                )

                            shared_valid = (
                                old_perfusion_valid
                                & new_perfusion_valid
                            )

                            comparison['shared_valid_voxels'] = int(
                                shared_valid.sum()
                            )
                            comparison['old_valid_new_invalid_voxels'] = int(
                                (
                                    old_perfusion_valid
                                    & ~new_perfusion_valid
                                ).sum()
                            )
                            comparison['new_valid_old_invalid_voxels'] = int(
                                (
                                    new_perfusion_valid
                                    & ~old_perfusion_valid
                                ).sum()
                            )

                            comparison[
                                'shared_valid_perfusion_channels_equal'
                            ] = bool(
                                shared_valid.any()
                                and all(
                                    np.array_equal(
                                        image[channel_index][shared_valid],
                                        old_image[channel_index][shared_valid],
                                    )
                                    for channel_index in (
                                        SHARED_VALID_PERFUSION_CHANNEL_INDICES
                                    )
                                )
                            )
                    except Exception as exc:
                        comparison['error'] = repr(exc)

                comparison_rows.append(comparison)

        case_record['valid'] = len(errors) == 0
        case_record['error'] = ' | '.join(errors) if errors else None

    except Exception as exc:
        case_record['error'] = repr(exc)

    case_rows.append(case_record)

    if case_index == 1 or case_index % 20 == 0 or case_index == EXPECTED_SUBJECTS:
        print(f'Verified {case_index}/{EXPECTED_SUBJECTS}: {subject_id}')

case_df = pd.DataFrame(case_rows)
channel_df = pd.DataFrame(channel_rows)
comparison_df = pd.DataFrame(comparison_rows)

case_df.to_csv(CASE_AUDIT_PATH, index=False)
channel_df.to_csv(CHANNEL_AUDIT_PATH, index=False)
comparison_df.to_csv(COMPARISON_PATH, index=False)

valid_count = int(case_df['valid'].sum())
invalid_df = case_df[~case_df['valid']].copy()

if not NEW_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Missing r1 manifest: {NEW_MANIFEST_PATH}'
    )

new_manifest_df = pd.read_csv(NEW_MANIFEST_PATH)
manifest_split_counts = new_manifest_df['dev_split'].value_counts().to_dict()

geometry_checks = [
    'target_binary_equal',
    'target_soft_equal',
    'cow_label_equal_after_schema_sanitization',
    'affine_equal',
]
comparison_complete = bool(
    len(comparison_df) == EXPECTED_SUBJECTS
    and comparison_df['old_file_exists'].all()
    and comparison_df['error'].isna().all()
)

geometry_ok = bool(
    comparison_complete
    and all(
        comparison_df[column].fillna(False).all()
        for column in geometry_checks
    )
)

structural_channels_ok = bool(
    comparison_complete
    and comparison_df[
        'structural_channels_equal'
    ].fillna(False).all()
)

shared_valid_perfusion_ok = bool(
    comparison_complete
    and comparison_df[
        'shared_valid_perfusion_channels_equal'
    ].fillna(False).all()
)

gate_passed = bool(
    valid_count == EXPECTED_SUBJECTS
    and len(new_manifest_df) == EXPECTED_SUBJECTS
    and manifest_split_counts == EXPECTED_SPLITS
    and geometry_ok
    and structural_channels_ok
    and shared_valid_perfusion_ok
)

gate = {
    'preprocessing_version': NEW_VERSION,
    'expected_subjects': EXPECTED_SUBJECTS,
    'valid_subjects': valid_count,
    'manifest_subjects': int(len(new_manifest_df)),
    'manifest_split_counts': manifest_split_counts,
    'comparison_complete': comparison_complete,
    'geometry_checks_passed': geometry_ok,
    'structural_channels': STRUCTURAL_CHANNEL_NAMES,
    'structural_channels_preserved': structural_channels_ok,
    'shared_valid_perfusion_channels': (
        SHARED_VALID_PERFUSION_CHANNEL_NAMES
    ),
    'shared_valid_perfusion_channels_preserved': (
        shared_valid_perfusion_ok
    ),
    'source_mtt_outside_configured_range_subjects': int(
        (case_df['source_mtt_invalid_voxels'].fillna(0) > 0).sum()
    ),
    'gate_passed': gate_passed,
    'case_audit_path': str(CASE_AUDIT_PATH),
    'channel_audit_path': str(CHANNEL_AUDIT_PATH),
    'comparison_path': str(COMPARISON_PATH),
    'manifest_sha256': file_sha256(NEW_MANIFEST_PATH),
}

GATE_PATH.write_text(json.dumps(gate, indent=2), encoding='utf-8')

print('\nCase validation counts:')
display(
    case_df['valid'].value_counts(dropna=False)
    .rename_axis('valid')
    .reset_index(name='subjects')
)

print('\nSplit counts:')
display(
    new_manifest_df.groupby('dev_split').size()
    .reset_index(name='subjects')
)

print('\nSource MTT voxels outside the configured physical range:')
print(
    'This includes expected background/edge voxels and is informational. '
    'It is not a case-level failure.'
)
display(
    case_df[case_df['source_mtt_invalid_voxels'].fillna(0) > 0][
        [
            'subject_id',
            'source_mtt_invalid_voxels',
            'source_mtt_valid_fraction',
        ]
    ].sort_values('subject_id')
)

print('\nPer-channel finite/range summary:')
display(
    channel_df.groupby(['channel_index', 'channel_name'], as_index=False)
    .agg(
        cases=('subject_id', 'count'),
        cases_with_nonfinite=(
            'nonfinite_voxels',
            lambda values: int((values > 0).sum()),
        ),
        global_min=('minimum', 'min'),
        global_max=('maximum', 'max'),
        mean_of_case_means=('mean', 'mean'),
    )
)

print('\nOriginal-versus-r1 comparison:')
display(
    pd.DataFrame({
        'check': [
            'Comparison complete',
            'Geometry and targets preserved',
            'Schema-safe CoW preserved',
            'Structural image channels preserved',
            'Perfusion values preserved on shared-valid voxels',
            'Final gate passed',
        ],
        'passed': [
            comparison_complete,
            bool(
                comparison_complete
                and comparison_df[
                    ['target_binary_equal', 'target_soft_equal', 'affine_equal']
                ].fillna(False).all().all()
            ),
            bool(
                comparison_complete
                and comparison_df[
                    'cow_label_equal_after_schema_sanitization'
                ].fillna(False).all()
            ),
            structural_channels_ok,
            shared_valid_perfusion_ok,
            gate_passed,
        ],
    })
)

print('\nPerfusion-validity transition summary:')
display(
    comparison_df[
        [
            'shared_valid_voxels',
            'old_valid_new_invalid_voxels',
            'new_valid_old_invalid_voxels',
        ]
    ].describe()
)

comparison_failure_mask = (
    ~comparison_df['target_binary_equal'].fillna(False)
    | ~comparison_df['target_soft_equal'].fillna(False)
    | ~comparison_df[
        'cow_label_equal_after_schema_sanitization'
    ].fillna(False)
    | ~comparison_df['affine_equal'].fillna(False)
    | ~comparison_df['structural_channels_equal'].fillna(False)
    | ~comparison_df[
        'shared_valid_perfusion_channels_equal'
    ].fillna(False)
    | comparison_df['error'].notna()
)

if comparison_failure_mask.any():
    print('\nFailed old-versus-r1 comparison rows:')
    display(comparison_df[comparison_failure_mask])

if len(invalid_df):
    print('\nInvalid r1 cases:')
    display(invalid_df[['subject_id', 'path', 'error']])

if not gate_passed:
    raise RuntimeError(
        'TRIAL8 V2 R1 VERIFICATION FAILED. Do not continue to atlas '
        'processing. Review the saved audits above.'
    )

frozen_manifest_df = new_manifest_df.copy()
case_metrics = case_df.set_index('subject_id')

frozen_manifest_df['vascufate_v2_target_binary_voxels'] = (
    frozen_manifest_df['subject_id'].map(
        case_metrics['target_binary_voxels']
    ).astype(np.int64)
)
frozen_manifest_df['vascufate_v2_target_soft_sum'] = (
    frozen_manifest_df['subject_id'].map(
        case_metrics['target_soft_sum']
    ).astype(np.float64)
)

if not np.isfinite(
    frozen_manifest_df['vascufate_v2_target_soft_sum']
).all():
    raise RuntimeError(
        'Refusing to freeze manifest because target-soft sums are nonfinite.'
    )

frozen_manifest_df.to_csv(FROZEN_MANIFEST_PATH, index=False)

print('\n' + '=' * 80)
print('TRIAL8 V2 R1 VERIFICATION PASSED')
print('=' * 80)
print(f'Valid subjects:          {valid_count}/{EXPECTED_SUBJECTS}')
print(f'Frozen manifest:         {FROZEN_MANIFEST_PATH}')
print(f'Case audit:              {CASE_AUDIT_PATH}')
print(f'Channel audit:           {CHANNEL_AUDIT_PATH}')
print(f'Geometry comparison:     {COMPARISON_PATH}')
print(f'Gate record:             {GATE_PATH}')
print('\nThe dataset is now eligible for vascular-atlas preflight.')


In [ ]:
import sys
import json
import re
import hashlib
import shutil
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
UNFROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1.csv'
)
VERIFICATION_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'final_verification'
    / 'r1_verification_gate.json'
)

ATLAS_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
)
SOURCE_PARENT = ATLAS_ROOT / 'source'
SOURCE_DIR = SOURCE_PARENT / 'Arterial_Atlas'
AUDIT_DIR = ATLAS_ROOT / 'acquisition_audit'

ATLAS_REPOSITORY_URL = 'https://github.com/Chin-Fu-Liu/Arterial_Atlas.git'
ATLAS_PROJECT_URL = 'https://www.nitrc.org/projects/arterialatlas/'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_ATLAS_182_SHAPE = (182, 218, 182)
EXPECTED_ATLAS_182_SPACING_MM = (1.0, 1.0, 1.0)


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


for path in [ATLAS_ROOT, SOURCE_PARENT, AUDIT_DIR]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def run_command(args, cwd=None):
    result = subprocess.run(
        [str(x) for x in args],
        cwd=None if cwd is None else str(cwd),
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, args))}\n"
            f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )
    return result.stdout.strip()






required_paths = [
    FROZEN_MANIFEST_PATH,
    UNFROZEN_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Required verified Trial8 files are missing:\n' + '\n'.join(missing_paths)
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('gate_passed') is not True:
    raise RuntimeError(
        'The Trial8 v2 r1 verification gate is not passed. '
        'Do not acquire or register the atlas yet.'
    )

expected_unfrozen_hash = verification_gate.get('manifest_sha256')
current_unfrozen_hash = file_sha256(UNFROZEN_MANIFEST_PATH)
if not expected_unfrozen_hash or current_unfrozen_hash != expected_unfrozen_hash:
    raise RuntimeError(
        'The verified Trial8 manifest changed after the verification gate was '
        'written. Rerun final verification before atlas work.'
    )

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Frozen manifest has {len(manifest_df)} subjects; '
        f'expected {EXPECTED_SUBJECTS}.'
    )

split_counts = {
    str(key): int(value)
    for key, value in manifest_df['dev_split'].value_counts().to_dict().items()
}
if split_counts != EXPECTED_SPLITS:
    raise RuntimeError(
        f'Frozen manifest split counts changed: {split_counts}; '
        f'expected {EXPECTED_SPLITS}.'
    )

if manifest_df['subject_id'].duplicated().any():
    duplicates = manifest_df.loc[
        manifest_df['subject_id'].duplicated(keep=False), 'subject_id'
    ].tolist()
    raise RuntimeError(f'Duplicate subjects in frozen manifest: {duplicates}')

print('Trial8 frozen-data gate passed.')
print(f'Frozen manifest subjects: {len(manifest_df)}')
print(f'Split counts: {split_counts}')
print(f'Frozen manifest SHA-256: {file_sha256(FROZEN_MANIFEST_PATH)}')






git_path = shutil.which('git')
if git_path is None:
    raise RuntimeError('git is unavailable in this Colab runtime.')

if not SOURCE_DIR.exists():
    local_temp_root = Path(tempfile.mkdtemp(prefix='trial8_atlas_clone_', dir='/tmp'))
    local_clone = local_temp_root / 'Arterial_Atlas'
    try:
        print(f'Cloning official atlas source: {ATLAS_REPOSITORY_URL}')
        run_command(
            [git_path, 'clone', '--depth', '1', ATLAS_REPOSITORY_URL, local_clone]
        )
        if SOURCE_DIR.exists():
            raise RuntimeError(
                f'Atlas destination appeared during acquisition: {SOURCE_DIR}'
            )
        shutil.move(str(local_clone), str(SOURCE_DIR))
    finally:
        if local_temp_root.exists():
            shutil.rmtree(local_temp_root)
    print(f'Saved atlas snapshot: {SOURCE_DIR}')
else:
    print(f'Using existing atlas snapshot without updating it: {SOURCE_DIR}')

if not (SOURCE_DIR / '.git').is_dir():
    raise RuntimeError(
        'Atlas source exists but is not a Git snapshot. Move it aside and rerun '
        'this cell so provenance can be captured safely.'
    )

repo_commit = run_command([git_path, 'rev-parse', 'HEAD'], cwd=SOURCE_DIR)
repo_branch = run_command(
    [git_path, 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=SOURCE_DIR
)
repo_remote = run_command(
    [git_path, 'config', '--get', 'remote.origin.url'], cwd=SOURCE_DIR
)
repo_dirty_lines = run_command(
    [git_path, 'status', '--porcelain', '--untracked-files=no'], cwd=SOURCE_DIR
)
repo_clean = repo_dirty_lines == ''

if repo_remote.rstrip('/') != ATLAS_REPOSITORY_URL.rstrip('/'):
    raise RuntimeError(
        f'Unexpected atlas Git remote: {repo_remote}\n'
        f'Expected: {ATLAS_REPOSITORY_URL}'
    )
if not repo_clean:
    raise RuntimeError(
        'The saved atlas source has tracked modifications. Refusing to continue '\
        'because the source is no longer an auditable upstream snapshot.'
    )

print(f'Atlas Git commit: {repo_commit}')
print(f'Atlas Git branch: {repo_branch}')
print('Atlas tracked source files are clean.')






nifti_paths = sorted(
    list(SOURCE_DIR.rglob('*.nii')) + list(SOURCE_DIR.rglob('*.nii.gz'))
)
lookup_paths = sorted(
    {
        path
        for pattern in ['*Label*.txt', '*label*.txt', '*Lable*.txt', '*lable*.txt']
        for path in SOURCE_DIR.rglob(pattern)
    }
)
license_paths = sorted(
    path for path in SOURCE_DIR.rglob('*')
    if path.is_file() and path.name.lower().startswith('license')
)

if not nifti_paths:
    raise RuntimeError('No NIfTI files were found in the atlas repository.')
if not license_paths:
    raise RuntimeError('No atlas license file was found in the repository.')


def is_atlas_182_path(path):
    normalized = str(path.relative_to(SOURCE_DIR)).replace('\\', '/').lower()
    return 'atlas_182' in normalized or 'atlas182' in normalized


atlas_182_paths = [path for path in nifti_paths if is_atlas_182_path(path)]
level2_candidates = [
    path for path in atlas_182_paths if 'level2' in path.name.lower()
]
fine_candidates = [
    path for path in atlas_182_paths
    if 'level2' not in path.name.lower()
    and 'prob' not in path.name.lower()
    and 'border' not in path.name.lower()
]

print('\nDiscovered atlas NIfTI files:')
for path in nifti_paths:
    print(' -', path.relative_to(SOURCE_DIR))

if len(fine_candidates) != 1 or len(level2_candidates) != 1:
    raise RuntimeError(
        'Could not uniquely resolve the deterministic Atlas_182 maps.\n'
        f'Fine candidates: {[str(p.relative_to(SOURCE_DIR)) for p in fine_candidates]}\n'
        f'Level-2 candidates: {[str(p.relative_to(SOURCE_DIR)) for p in level2_candidates]}'
    )

fine_atlas_path = fine_candidates[0]
level2_atlas_path = level2_candidates[0]


def nifti_audit_row(path, role):
    image = nib.load(str(path))
    raw_shape = tuple(int(v) for v in image.shape)

    
    
    
    
    if len(raw_shape) == 3:
        spatial_shape = raw_shape
        singleton_fourth_dimension = False
        data = np.asanyarray(image.dataobj)
    elif len(raw_shape) == 4 and raw_shape[3] == 1:
        spatial_shape = raw_shape[:3]
        singleton_fourth_dimension = True
        data = np.asanyarray(image.dataobj)[..., 0]
    else:
        spatial_shape = raw_shape
        singleton_fourth_dimension = False
        data = np.asanyarray(image.dataobj)

    effectively_3d = len(spatial_shape) == 3
    finite_mask = np.isfinite(data)
    all_finite = bool(finite_mask.all())

    if all_finite:
        rounded = np.rint(data)
        max_integer_error = float(np.max(np.abs(data - rounded)))
        integer_like = bool(max_integer_error <= 1e-4)
        unique_values = np.unique(rounded.astype(np.int64)) if integer_like else np.unique(data)
        data_min = float(np.min(data))
        data_max = float(np.max(data))
        nonzero_voxels = int(np.count_nonzero(data))
    else:
        max_integer_error = float('nan')
        integer_like = False
        unique_values = np.array([])
        data_min = float('nan')
        data_max = float('nan')
        nonzero_voxels = 0

    qform, qform_code = image.get_qform(coded=True)
    sform, sform_code = image.get_sform(coded=True)
    del qform, sform

    return {
        'role': role,
        'relative_path': str(path.relative_to(SOURCE_DIR)),
        'raw_shape': raw_shape,
        'raw_ndim': int(len(raw_shape)),
        'spatial_shape': spatial_shape,
        'effectively_3d': effectively_3d,
        'singleton_fourth_dimension': singleton_fourth_dimension,
        'dtype': str(image.get_data_dtype()),
        'orientation': ''.join(nib.aff2axcodes(image.affine)),
        'spacing_mm': tuple(float(v) for v in image.header.get_zooms()[:3]),
        'qform_code': int(qform_code or 0),
        'sform_code': int(sform_code or 0),
        'all_finite': all_finite,
        'nonfinite_voxels': int(data.size - finite_mask.sum()),
        'integer_like': integer_like,
        'max_integer_error': max_integer_error,
        'min': data_min,
        'max': data_max,
        'unique_value_count': int(len(unique_values)),
        'positive_label_count': int(np.sum(unique_values > 0)),
        'unique_values': [
            int(value) if integer_like else float(value)
            for value in unique_values.tolist()
        ],
        'nonzero_voxels': nonzero_voxels,
        'affine': np.asarray(image.affine, dtype=float).tolist(),
        'sha256': file_sha256(path),
        'size_bytes': int(path.stat().st_size),
    }


fine_row = nifti_audit_row(fine_atlas_path, 'fine_territories')
level2_row = nifti_audit_row(level2_atlas_path, 'major_territories_level2')
atlas_rows = [fine_row, level2_row]

fine_image = nib.load(str(fine_atlas_path))
level2_image = nib.load(str(level2_atlas_path))
same_raw_shape = tuple(fine_image.shape) == tuple(level2_image.shape)
same_spatial_shape = (
    tuple(fine_row['spatial_shape']) == tuple(level2_row['spatial_shape'])
)
same_affine = bool(
    np.allclose(fine_image.affine, level2_image.affine, atol=1e-5, rtol=1e-6)
)
max_affine_difference = float(
    np.max(np.abs(fine_image.affine - level2_image.affine))
)

audit_display_df = pd.DataFrame(atlas_rows).drop(
    columns=['affine', 'unique_values']
)
print('\nSelected deterministic Atlas_182 maps:')
display(audit_display_df)

print('\nFine-atlas unique labels:')
print(fine_row['unique_values'])
print('\nLevel-2 unique labels:')
print(level2_row['unique_values'])
print(f'\nFine/level-2 raw shape match: {same_raw_shape}')
print(f'Fine/level-2 spatial shape match: {same_spatial_shape}')
print(f'Fine/level-2 affine match: {same_affine}')
print(f'Max affine difference: {max_affine_difference:.8g}')






lookup_records = []
lookup_preview_parts = []

for path in lookup_paths:
    text = path.read_text(encoding='utf-8', errors='replace')
    nonempty_lines = [line for line in text.splitlines() if line.strip()]
    numeric_line_count = sum(
        bool(re.match(r'^\s*\d+\b', line)) for line in nonempty_lines
    )
    record = {
        'relative_path': str(path.relative_to(SOURCE_DIR)),
        'sha256': file_sha256(path),
        'size_bytes': int(path.stat().st_size),
        'nonempty_lines': int(len(nonempty_lines)),
        'lines_starting_with_integer': int(numeric_line_count),
    }
    lookup_records.append(record)
    lookup_preview_parts.append(
        f"===== {record['relative_path']} =====\n" +
        '\n'.join(nonempty_lines[:120])
    )

lookup_preview_path = AUDIT_DIR / 'atlas_label_lookup_preview.txt'
lookup_preview_path.write_text(
    '\n\n'.join(lookup_preview_parts) if lookup_preview_parts else 'No lookup files found.\n',
    encoding='utf-8',
)

print('\nDiscovered label lookup files:')
if lookup_records:
    display(pd.DataFrame(lookup_records))
else:
    print('None found.')







def tuple_close(values, expected, atol=1e-5):
    return bool(np.allclose(np.asarray(values), np.asarray(expected), atol=atol, rtol=0))


source_checks = {
    'trial8_verification_gate_passed': verification_gate.get('gate_passed') is True,
    'frozen_manifest_subject_count_ok': len(manifest_df) == EXPECTED_SUBJECTS,
    'frozen_manifest_split_counts_ok': split_counts == EXPECTED_SPLITS,
    'repository_remote_ok': repo_remote.rstrip('/') == ATLAS_REPOSITORY_URL.rstrip('/'),
    'repository_clean': repo_clean,
    'license_present': len(license_paths) >= 1,
    'lookup_present': len(lookup_paths) >= 1,
    'fine_atlas_is_effectively_3d': fine_row['effectively_3d'],
    'level2_atlas_is_effectively_3d': level2_row['effectively_3d'],
    'fine_atlas_spatial_shape_ok': (
        tuple(fine_row['spatial_shape']) == EXPECTED_ATLAS_182_SHAPE
    ),
    'level2_atlas_spatial_shape_ok': (
        tuple(level2_row['spatial_shape']) == EXPECTED_ATLAS_182_SHAPE
    ),
    'fine_atlas_spacing_ok': tuple_close(
        fine_row['spacing_mm'], EXPECTED_ATLAS_182_SPACING_MM
    ),
    'level2_atlas_spacing_ok': tuple_close(
        level2_row['spacing_mm'], EXPECTED_ATLAS_182_SPACING_MM
    ),
    'fine_atlas_finite': fine_row['all_finite'],
    'level2_atlas_finite': level2_row['all_finite'],
    'fine_atlas_integer_like': fine_row['integer_like'],
    'level2_atlas_integer_like': level2_row['integer_like'],
    'fine_atlas_has_at_least_30_positive_labels': fine_row['positive_label_count'] >= 30,
    'level2_has_at_least_4_positive_labels': level2_row['positive_label_count'] >= 4,
    'fine_and_level2_raw_shapes_match': same_raw_shape,
    'fine_and_level2_spatial_shapes_match': same_spatial_shape,
    'fine_and_level2_affines_match': same_affine,
}
source_gate_passed = bool(all(source_checks.values()))

atlas_audit_csv_path = AUDIT_DIR / 'atlas_source_audit.csv'
atlas_lookup_csv_path = AUDIT_DIR / 'atlas_lookup_files.csv'
atlas_provenance_path = AUDIT_DIR / 'atlas_source_provenance.json'
atlas_gate_path = AUDIT_DIR / 'atlas_acquisition_gate.json'

pd.DataFrame(atlas_rows).assign(
    unique_values=lambda frame: frame['unique_values'].map(json.dumps),
    affine=lambda frame: frame['affine'].map(json.dumps),
).to_csv(atlas_audit_csv_path, index=False)

pd.DataFrame(lookup_records).to_csv(atlas_lookup_csv_path, index=False)

license_records = [
    {
        'relative_path': str(path.relative_to(SOURCE_DIR)),
        'sha256': file_sha256(path),
        'size_bytes': int(path.stat().st_size),
    }
    for path in license_paths
]

provenance = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'trial8_preprocessing_version': PREPROCESSING_VERSION,
    'trial8_frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'trial8_frozen_manifest_sha256': file_sha256(FROZEN_MANIFEST_PATH),
    'trial8_verification_gate_path': str(VERIFICATION_GATE_PATH),
    'trial8_verification_gate_sha256': file_sha256(VERIFICATION_GATE_PATH),
    'atlas_repository_url': ATLAS_REPOSITORY_URL,
    'atlas_project_url': ATLAS_PROJECT_URL,
    'atlas_source_directory': str(SOURCE_DIR),
    'atlas_git_commit': repo_commit,
    'atlas_git_branch': repo_branch,
    'atlas_git_remote': repo_remote,
    'atlas_git_clean': repo_clean,
    'selected_fine_atlas': fine_row,
    'selected_level2_atlas': level2_row,
    'lookup_files': lookup_records,
    'license_files': license_records,
    'registration_performed': False,
    'patient_data_modified': False,
}
atlas_provenance_path.write_text(
    json.dumps(provenance, indent=2), encoding='utf-8'
)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'gate_passed': source_gate_passed,
    'checks': source_checks,
    'atlas_git_commit': repo_commit,
    'fine_atlas_path': str(fine_atlas_path),
    'level2_atlas_path': str(level2_atlas_path),
    'source_audit_path': str(atlas_audit_csv_path),
    'source_provenance_path': str(atlas_provenance_path),
    'label_lookup_preview_path': str(lookup_preview_path),
    'registration_performed': False,
}
atlas_gate_path.write_text(json.dumps(gate_record, indent=2), encoding='utf-8')

print('\nAtlas acquisition checks:')
display(
    pd.DataFrame(
        [{'check': key, 'passed': value} for key, value in source_checks.items()]
    )
)

if not source_gate_passed:
    failed_checks = [key for key, value in source_checks.items() if not value]
    raise RuntimeError(
        'TRIAL8 ATLAS ACQUISITION AUDIT FAILED. Do not begin registration. '
        f'Failed checks: {failed_checks}'
    )

print('\n' + '=' * 80)
print('TRIAL8 ATLAS ACQUISITION AND SOURCE AUDIT PASSED')
print('=' * 80)
print(f'Atlas commit:          {repo_commit}')
print(f'Fine atlas:            {fine_atlas_path}')
print(f'Level-2 atlas:         {level2_atlas_path}')
print(f'Provenance record:     {atlas_provenance_path}')
print(f'Acquisition gate:      {atlas_gate_path}')
print(f'Lookup preview:        {lookup_preview_path}')
print('\nNo subject registration was performed.')
print('Stop here and check this output before running registration preflight.')


In [ ]:
import sys
import json
import re
import hashlib
import shutil
import subprocess
import tempfile
import urllib.request
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'

ATLAS_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
)
SOURCE_DIR = ATLAS_ROOT / 'source' / 'Arterial_Atlas'
ACQUISITION_DIR = ATLAS_ROOT / 'acquisition_audit'
LABEL_AUDIT_DIR = ATLAS_ROOT / 'label_hierarchy_audit'
REFERENCE_DIR = ATLAS_ROOT / 'reference' / 'JHU_MNI_SS'

ACQUISITION_GATE_PATH = ACQUISITION_DIR / 'atlas_acquisition_gate.json'
FINE_ATLAS_PATH = SOURCE_DIR / 'data' / 'Atlas_182' / 'ArterialAtlas.nii'
LEVEL2_ATLAS_PATH = (
    SOURCE_DIR / 'data' / 'Atlas_182' / 'ArterialAtlas_level2.nii'
)
LOOKUP_PATH = SOURCE_DIR / 'data' / 'ArterialAtlasLables.txt'

JHU_REFERENCE_REPOSITORY = 'https://github.com/muschellij2/Eve_Atlas.git'
JHU_REFERENCE_BRANCH = 'master'
JHU_REFERENCE_FILENAME = 'JHU_MNI_SS_T1_brain_182x218x182.nii.gz'
JHU_REFERENCE_PATH = REFERENCE_DIR / JHU_REFERENCE_FILENAME
JHU_REFERENCE_PROVENANCE_PATH = REFERENCE_DIR / 'source_provenance.json'




EXPECTED_SHAPE = (182, 218, 182)
FSL_MNI152_AFFINE = np.array(
    [
        [-1.0, 0.0, 0.0, 90.0],
        [0.0, 1.0, 0.0, -126.0],
        [0.0, 0.0, 1.0, -72.0],
        [0.0, 0.0, 0.0, 1.0],
    ],
    dtype=np.float64,
)


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


for path in [LABEL_AUDIT_DIR, REFERENCE_DIR]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def run_command(args, cwd=None):
    result = subprocess.run(
        [str(value) for value in args],
        cwd=None if cwd is None else str(cwd),
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, args))}\n"
            f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )
    return result.stdout.strip()


required_paths = [
    ACQUISITION_GATE_PATH,
    FINE_ATLAS_PATH,
    LEVEL2_ATLAS_PATH,
    LOOKUP_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required atlas files:\n' + '\n'.join(missing))

acquisition_gate = json.loads(
    ACQUISITION_GATE_PATH.read_text(encoding='utf-8')
)
if acquisition_gate.get('gate_passed') is not True:
    raise RuntimeError(
        'Atlas acquisition gate is not passed. Do not continue to label audit.'
    )

expected_commit = 'fbeb3fe70b6fb8185244d02f9c6b6e07c13235e0'
if acquisition_gate.get('atlas_git_commit') != expected_commit:
    raise RuntimeError(
        'Atlas commit differs from the audited source snapshot. '
        f"Found {acquisition_gate.get('atlas_git_commit')}; expected {expected_commit}."
    )

print('Atlas acquisition gate passed.')
print(f'Pinned atlas commit: {expected_commit}')







if JHU_REFERENCE_PATH.exists() or JHU_REFERENCE_PROVENANCE_PATH.exists():
    if not JHU_REFERENCE_PATH.exists() or not JHU_REFERENCE_PROVENANCE_PATH.exists():
        raise RuntimeError(
            'The JHU reference or its provenance record is missing. Move the '
            'incomplete reference directory aside and rerun this cell.'
        )

    jhu_provenance = json.loads(
        JHU_REFERENCE_PROVENANCE_PATH.read_text(encoding='utf-8')
    )
    expected_jhu_hash = jhu_provenance.get('sha256')
    current_jhu_hash = file_sha256(JHU_REFERENCE_PATH)
    if not expected_jhu_hash or current_jhu_hash != expected_jhu_hash:
        raise RuntimeError(
            'The saved JHU_MNI_SS reference no longer matches its recorded '
            'SHA-256 hash.'
        )
    jhu_commit = jhu_provenance['git_commit']
    jhu_download_url = jhu_provenance['download_url']
    print(f'Using existing pinned JHU_MNI_SS reference: {JHU_REFERENCE_PATH}')
else:
    git_path = shutil.which('git')
    if git_path is None:
        raise RuntimeError('git is unavailable in this Colab runtime.')

    ls_remote = run_command(
        [
            git_path,
            'ls-remote',
            JHU_REFERENCE_REPOSITORY,
            f'refs/heads/{JHU_REFERENCE_BRANCH}',
        ]
    )
    if not ls_remote:
        raise RuntimeError('Could not resolve the JHU_MNI_SS source commit.')
    jhu_commit = ls_remote.split()[0]
    if not re.fullmatch(r'[0-9a-f]{40}', jhu_commit):
        raise RuntimeError(f'Unexpected JHU source commit value: {jhu_commit}')

    jhu_download_url = (
        'https://raw.githubusercontent.com/muschellij2/Eve_Atlas/'
        f'{jhu_commit}/{JHU_REFERENCE_FILENAME}'
    )
    local_temp_dir = Path(
        tempfile.mkdtemp(prefix='trial8_jhu_reference_', dir='/tmp')
    )
    local_download = local_temp_dir / JHU_REFERENCE_FILENAME
    try:
        print(f'Downloading pinned JHU_MNI_SS reference: {jhu_download_url}')
        urllib.request.urlretrieve(jhu_download_url, local_download)
        if not local_download.exists() or local_download.stat().st_size == 0:
            raise RuntimeError('JHU_MNI_SS reference download is empty.')
        shutil.move(str(local_download), str(JHU_REFERENCE_PATH))
    finally:
        if local_temp_dir.exists():
            shutil.rmtree(local_temp_dir)

    jhu_provenance = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'repository': JHU_REFERENCE_REPOSITORY,
        'branch_at_first_acquisition': JHU_REFERENCE_BRANCH,
        'git_commit': jhu_commit,
        'download_url': jhu_download_url,
        'filename': JHU_REFERENCE_FILENAME,
        'path': str(JHU_REFERENCE_PATH),
        'sha256': file_sha256(JHU_REFERENCE_PATH),
        'size_bytes': int(JHU_REFERENCE_PATH.stat().st_size),
        'auto_update_disabled': True,
    }
    JHU_REFERENCE_PROVENANCE_PATH.write_text(
        json.dumps(jhu_provenance, indent=2), encoding='utf-8'
    )

print(f'JHU_MNI_SS source commit: {jhu_commit}')
print(f'JHU_MNI_SS SHA-256: {file_sha256(JHU_REFERENCE_PATH)}')


def load_effective_label_volume(path):
    image = nib.load(str(path))
    data = np.asanyarray(image.dataobj)
    raw_shape = tuple(int(v) for v in data.shape)

    if data.ndim == 4 and data.shape[-1] == 1:
        data = data[..., 0]
    elif data.ndim != 3:
        raise RuntimeError(f'Unsupported label-map shape for {path}: {raw_shape}')

    if not np.isfinite(data).all():
        raise RuntimeError(f'Nonfinite atlas values found in {path}')

    integer_error = float(np.max(np.abs(data - np.rint(data))))
    if integer_error > 1e-4:
        raise RuntimeError(
            f'Atlas values are not effectively integral in {path}: '
            f'maximum error {integer_error}'
        )

    labels = np.rint(data).astype(np.int16)
    return image, labels, raw_shape


def load_reference_volume(path):
    image = nib.load(str(path))
    data = np.asanyarray(image.dataobj)
    raw_shape = tuple(int(value) for value in data.shape)
    if data.ndim == 4 and data.shape[-1] == 1:
        data = data[..., 0]
    elif data.ndim != 3:
        raise RuntimeError(f'Unsupported reference shape for {path}: {raw_shape}')
    if not np.isfinite(data).all():
        raise RuntimeError(f'Nonfinite values found in reference image: {path}')
    return image, data.astype(np.float32), raw_shape


fine_image, fine_labels, fine_raw_shape = load_effective_label_volume(
    FINE_ATLAS_PATH
)
level2_image, level2_labels, level2_raw_shape = load_effective_label_volume(
    LEVEL2_ATLAS_PATH
)
jhu_image, jhu_data, jhu_raw_shape = load_reference_volume(JHU_REFERENCE_PATH)

fine_shape = tuple(int(v) for v in fine_labels.shape)
level2_shape = tuple(int(v) for v in level2_labels.shape)
jhu_shape = tuple(int(v) for v in jhu_data.shape)
fine_jhu_affine_error = float(
    np.max(np.abs(fine_image.affine - jhu_image.affine))
)
level2_jhu_affine_error = float(
    np.max(np.abs(level2_image.affine - jhu_image.affine))
)
fine_fsl_affine_error = float(
    np.max(np.abs(fine_image.affine - FSL_MNI152_AFFINE))
)
jhu_fsl_affine_error = float(
    np.max(np.abs(jhu_image.affine - FSL_MNI152_AFFINE))
)

geometry_checks = {
    'jhu_reference_shape_is_182x218x182': jhu_shape == EXPECTED_SHAPE,
    'fine_effective_shape_matches_jhu_reference': fine_shape == jhu_shape,
    'level2_effective_shape_matches_jhu_reference': level2_shape == jhu_shape,
    'fine_affine_matches_jhu_reference': fine_jhu_affine_error <= 1e-5,
    'level2_affine_matches_jhu_reference': level2_jhu_affine_error <= 1e-5,
    'fine_and_level2_affines_match': bool(
        np.allclose(fine_image.affine, level2_image.affine, atol=1e-5, rtol=1e-6)
    ),
    'jhu_reference_has_nonzero_signal': bool(np.count_nonzero(jhu_data) > 0),
}

print('\nAtlas and documented JHU_MNI_SS reference geometry:')
print(f'Fine raw shape:       {fine_raw_shape}')
print(f'Fine spatial shape:   {fine_shape}')
print(f'Level-2 raw shape:    {level2_raw_shape}')
print(f'Level-2 spatial shape:{level2_shape}')
print(f'JHU raw shape:        {jhu_raw_shape}')
print(f'JHU spatial shape:    {jhu_shape}')
print(f'Atlas orientation:    {"".join(nib.aff2axcodes(fine_image.affine))}')
print(f'JHU orientation:      {"".join(nib.aff2axcodes(jhu_image.affine))}')
print(f'Fine/JHU affine error:{fine_jhu_affine_error:.8g}')
print(f'Level2/JHU error:     {level2_jhu_affine_error:.8g}')
print('\nAtlas affine:')
print(np.asarray(fine_image.affine))
print('\nJHU_MNI_SS reference affine:')
print(np.asarray(jhu_image.affine))

print('\nInformational FSL-header comparison (not a gate):')
print(f'Atlas versus FSL affine error: {fine_fsl_affine_error:.8g}')
print(f'JHU versus FSL affine error:   {jhu_fsl_affine_error:.8g}')
print(
    'Shape compatibility with FSL does not imply that the published atlas '
    'uses the FSL world-coordinate header.'
)

print('\nGeometry checks:')
display(
    pd.DataFrame(
        [{'check': key, 'passed': value} for key, value in geometry_checks.items()]
    )
)

if not all(geometry_checks.values()):
    failed = [key for key, value in geometry_checks.items() if not value]
    raise RuntimeError(
        'Atlas does not match its documented JHU_MNI_SS intensity reference. '
        f'Do not begin registration. Failed checks: {failed}'
    )



lookup_text = LOOKUP_PATH.read_text(encoding='utf-8', errors='replace')
lookup_lines = [line.rstrip() for line in lookup_text.splitlines()]

print('\n' + '=' * 80)
print('AUTHORITATIVE ATLAS LOOKUP TABLE')
print('=' * 80)
for line_number, line in enumerate(lookup_lines, start=1):
    print(f'{line_number:03d}: {line}')

lookup_line_rows = []
for line_number, line in enumerate(lookup_lines, start=1):
    if not line.strip():
        continue
    integers = [int(value) for value in re.findall(r'(?<![\d.])-?\d+(?![\d.])', line)]
    lookup_line_rows.append(
        {
            'line_number': line_number,
            'text': line,
            'integer_tokens': json.dumps(integers),
        }
    )

lookup_lines_path = LABEL_AUDIT_DIR / 'atlas_lookup_lines.csv'
pd.DataFrame(lookup_line_rows).to_csv(lookup_lines_path, index=False)




fine_values = sorted(int(v) for v in np.unique(fine_labels) if v > 0)
level2_values = sorted(int(v) for v in np.unique(level2_labels) if v > 0)

hierarchy_rows = []
for fine_label in fine_values:
    mask = fine_labels == fine_label
    child_voxels = int(mask.sum())
    parents, counts = np.unique(level2_labels[mask], return_counts=True)
    order = np.argsort(counts)[::-1]
    parents = parents[order]
    counts = counts[order]

    positive_parent_pairs = [
        (int(parent), int(count))
        for parent, count in zip(parents, counts)
        if int(parent) > 0
    ]
    dominant_parent = positive_parent_pairs[0][0] if positive_parent_pairs else 0
    dominant_count = positive_parent_pairs[0][1] if positive_parent_pairs else 0
    parent_purity = dominant_count / child_voxels if child_voxels else 0.0

    hierarchy_rows.append(
        {
            'fine_label': fine_label,
            'fine_voxels': child_voxels,
            'dominant_level2_label': dominant_parent,
            'dominant_level2_voxels': dominant_count,
            'dominant_parent_purity': float(parent_purity),
            'overlapping_level2_labels': json.dumps(positive_parent_pairs),
            'background_overlap_voxels': int(counts[parents == 0].sum())
            if np.any(parents == 0)
            else 0,
        }
    )

hierarchy_df = pd.DataFrame(hierarchy_rows)
hierarchy_path = LABEL_AUDIT_DIR / 'fine_to_level2_hierarchy.csv'
hierarchy_df.to_csv(hierarchy_path, index=False)

print('\nFine-to-level-2 hierarchy derived from voxel overlap:')
display(hierarchy_df)

print('\nHierarchy summary:')
print(f'Fine positive labels: {fine_values}')
print(f'Level-2 positive labels: {level2_values}')
print(
    'Fine labels with exactly pure positive parent: '
    f"{int((hierarchy_df['dominant_parent_purity'] == 1.0).sum())}/"
    f'{len(hierarchy_df)}'
)
print(
    'Fine labels with any background overlap: '
    f"{int((hierarchy_df['background_overlap_voxels'] > 0).sum())}"
)

audit_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'atlas_git_commit': expected_commit,
    'fine_atlas_path': str(FINE_ATLAS_PATH),
    'fine_atlas_sha256': file_sha256(FINE_ATLAS_PATH),
    'level2_atlas_path': str(LEVEL2_ATLAS_PATH),
    'level2_atlas_sha256': file_sha256(LEVEL2_ATLAS_PATH),
    'lookup_path': str(LOOKUP_PATH),
    'lookup_sha256': file_sha256(LOOKUP_PATH),
    'jhu_reference_repository': JHU_REFERENCE_REPOSITORY,
    'jhu_reference_git_commit': jhu_commit,
    'jhu_reference_download_url': jhu_download_url,
    'jhu_reference_path': str(JHU_REFERENCE_PATH),
    'jhu_reference_sha256': file_sha256(JHU_REFERENCE_PATH),
    'geometry_checks': geometry_checks,
    'informational_fsl_header_comparison': {
        'atlas_max_affine_difference': fine_fsl_affine_error,
        'jhu_reference_max_affine_difference': jhu_fsl_affine_error,
        'used_as_gate': False,
    },
    'fine_positive_labels': fine_values,
    'level2_positive_labels': level2_values,
    'hierarchy_csv': str(hierarchy_path),
    'lookup_lines_csv': str(lookup_lines_path),
    'registration_performed': False,
    'patient_data_modified': False,
}
audit_record_path = LABEL_AUDIT_DIR / 'atlas_label_hierarchy_audit.json'
audit_record_path.write_text(json.dumps(audit_record, indent=2), encoding='utf-8')

print('\n' + '=' * 80)
print('TRIAL8 ATLAS LABEL AND HIERARCHY AUDIT COMPLETE')
print('=' * 80)
print(f'Lookup lines:     {lookup_lines_path}')
print(f'Hierarchy table:  {hierarchy_path}')
print(f'Audit record:     {audit_record_path}')
print('\nNo subject registration was performed.')
print('Stop here and check the lookup table and hierarchy output.')


In [ ]:
import sys
import json
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
ATLAS_COMMIT = 'fbeb3fe70b6fb8185244d02f9c6b6e07c13235e0'
SCHEMA_VERSION = 'vascular_hierarchy_v1'

ATLAS_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
)
SOURCE_DIR = ATLAS_ROOT / 'source' / 'Arterial_Atlas'
ACQUISITION_GATE_PATH = (
    ATLAS_ROOT / 'acquisition_audit' / 'atlas_acquisition_gate.json'
)
LABEL_AUDIT_PATH = (
    ATLAS_ROOT
    / 'label_hierarchy_audit'
    / 'atlas_label_hierarchy_audit.json'
)
SCHEMA_DIR = ATLAS_ROOT / 'schema' / SCHEMA_VERSION

FINE_ATLAS_PATH = SOURCE_DIR / 'data' / 'Atlas_182' / 'ArterialAtlas.nii'
PUBLISHED_LEVEL2_PATH = (
    SOURCE_DIR / 'data' / 'Atlas_182' / 'ArterialAtlas_level2.nii'
)
LOOKUP_PATH = SOURCE_DIR / 'data' / 'ArterialAtlasLables.txt'

SCHEMA_CSV_PATH = SCHEMA_DIR / 'vascular_label_schema.csv'
SCHEMA_JSON_PATH = SCHEMA_DIR / 'vascular_label_schema.json'
CANONICAL_LEVEL2_PATH = (
    SCHEMA_DIR / 'ArterialAtlas_level2_canonical_from_fine.nii.gz'
)
VASCULAR_ONLY_FINE_PATH = (
    SCHEMA_DIR / 'ArterialAtlas_fine_vascular_only.nii.gz'
)
MISMATCH_CSV_PATH = SCHEMA_DIR / 'published_vs_canonical_level2_mismatches.csv'
SCHEMA_GATE_PATH = SCHEMA_DIR / 'schema_freeze_gate.json'


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(SCHEMA_DIR)
SCHEMA_DIR.mkdir(parents=True, exist_ok=True)


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


required_paths = [
    ACQUISITION_GATE_PATH,
    LABEL_AUDIT_PATH,
    FINE_ATLAS_PATH,
    PUBLISHED_LEVEL2_PATH,
    LOOKUP_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required atlas files:\n' + '\n'.join(missing))

acquisition_gate = json.loads(
    ACQUISITION_GATE_PATH.read_text(encoding='utf-8')
)
label_audit = json.loads(LABEL_AUDIT_PATH.read_text(encoding='utf-8'))

if acquisition_gate.get('gate_passed') is not True:
    raise RuntimeError('Atlas acquisition gate is not passed.')
if acquisition_gate.get('atlas_git_commit') != ATLAS_COMMIT:
    raise RuntimeError('Atlas acquisition commit differs from the frozen commit.')
if not all(label_audit.get('geometry_checks', {}).values()):
    raise RuntimeError('Atlas-to-JHU reference geometry audit is not passed.')

if file_sha256(FINE_ATLAS_PATH) != label_audit.get('fine_atlas_sha256'):
    raise RuntimeError('Fine atlas changed after the hierarchy audit.')
if file_sha256(PUBLISHED_LEVEL2_PATH) != label_audit.get('level2_atlas_sha256'):
    raise RuntimeError('Published level-2 atlas changed after the hierarchy audit.')
if file_sha256(LOOKUP_PATH) != label_audit.get('lookup_sha256'):
    raise RuntimeError('Atlas lookup changed after the hierarchy audit.')

print('Atlas acquisition and JHU-reference geometry gates passed.')
print(f'Pinned atlas commit: {ATLAS_COMMIT}')


def load_label_volume(path):
    image = nib.load(str(path))
    data = np.asanyarray(image.dataobj)
    if data.ndim == 4 and data.shape[-1] == 1:
        data = data[..., 0]
    elif data.ndim != 3:
        raise RuntimeError(f'Unsupported atlas shape: {path} -> {data.shape}')
    if not np.isfinite(data).all():
        raise RuntimeError(f'Nonfinite atlas values: {path}')
    error = float(np.max(np.abs(data - np.rint(data))))
    if error > 1e-4:
        raise RuntimeError(f'Nonintegral atlas labels: {path}, error={error}')
    return image, np.rint(data).astype(np.int16)


fine_image, fine_labels = load_label_volume(FINE_ATLAS_PATH)
level2_image, published_level2 = load_label_volume(PUBLISHED_LEVEL2_PATH)

if fine_labels.shape != published_level2.shape:
    raise RuntimeError('Fine and published level-2 atlas shapes differ.')
if not np.allclose(fine_image.affine, level2_image.affine, atol=1e-5, rtol=1e-6):
    raise RuntimeError('Fine and published level-2 atlas affines differ.')





rows = [
    (1, 'anterior cerebral artery left', 'ACAL', 'left', 1, 'ACAL', 'ACA', 'Anterior', True),
    (2, 'anterior cerebral artery right', 'ACAR', 'right', 2, 'ACAR', 'ACA', 'Anterior', True),
    (3, 'medial lenticulostriate left', 'MLSL', 'left', 1, 'ACAL', 'ACA', 'Anterior', True),
    (4, 'medial lenticulostriate right', 'MLSR', 'right', 2, 'ACAR', 'ACA', 'Anterior', True),
    (5, 'lateral lenticulostriate left', 'LLSL', 'left', 3, 'MCAL', 'MCA', 'Anterior', True),
    (6, 'lateral lenticulostriate right', 'LLSR', 'right', 4, 'MCAR', 'MCA', 'Anterior', True),
    (7, 'frontal part of middle cerebral artery left', 'MCAFL', 'left', 3, 'MCAL', 'MCA', 'Anterior', True),
    (8, 'frontal part of middle cerebral artery right', 'MCAFR', 'right', 4, 'MCAR', 'MCA', 'Anterior', True),
    (9, 'parietal part of middle cerebral artery left', 'MCAPL', 'left', 3, 'MCAL', 'MCA', 'Anterior', True),
    (10, 'parietal part of middle cerebral artery right', 'MCAPR', 'right', 4, 'MCAR', 'MCA', 'Anterior', True),
    (11, 'temporal part of middle cerebral artery left', 'MCATL', 'left', 3, 'MCAL', 'MCA', 'Anterior', True),
    (12, 'temporal part of middle cerebral artery right', 'MCATR', 'right', 4, 'MCAR', 'MCA', 'Anterior', True),
    (13, 'occipital part of middle cerebral artery left', 'MCAOL', 'left', 3, 'MCAL', 'MCA', 'Anterior', True),
    (14, 'occipital part of middle cerebral artery right', 'MCAOR', 'right', 4, 'MCAR', 'MCA', 'Anterior', True),
    (15, 'insular part of middle cerebral artery left', 'MCAIL', 'left', 3, 'MCAL', 'MCA', 'Anterior', True),
    (16, 'insular part of middle cerebral artery right', 'MCAIR', 'right', 4, 'MCAR', 'MCA', 'Anterior', True),
    (17, 'temporal part of posterior cerebral artery left', 'PCATL', 'left', 5, 'PCAL', 'PCA', 'Posterior', True),
    (18, 'temporal part of posterior cerebral artery right', 'PCATR', 'right', 6, 'PCAR', 'PCA', 'Posterior', True),
    (19, 'occipital part of posterior cerebral artery left', 'PCAOL', 'left', 5, 'PCAL', 'PCA', 'Posterior', True),
    (20, 'occipital part of posterior cerebral artery right', 'PCAOR', 'right', 6, 'PCAR', 'PCA', 'Posterior', True),
    (21, 'posterior choroidal and thalamoperforators left', 'PCTPL', 'left', 5, 'PCAL', 'PCA', 'Posterior', True),
    (22, 'posterior choroidal and thalamoperforators right', 'PCTPR', 'right', 6, 'PCAR', 'PCA', 'Posterior', True),
    (23, 'anterior choroidal and thalamoperforators left', 'ACTPL', 'left', 5, 'PCAL', 'PCA', 'Posterior', True),
    (24, 'anterior choroidal and thalamoperforators right', 'ACTPR', 'right', 6, 'PCAR', 'PCA', 'Posterior', True),
    (25, 'basilar left', 'BL', 'left', 7, 'VBL', 'VB', 'Posterior', True),
    (26, 'basilar right', 'BR', 'right', 8, 'VBR', 'VB', 'Posterior', True),
    (27, 'superior cerebellar left', 'SCL', 'left', 7, 'VBL', 'VB', 'Posterior', True),
    (28, 'superior cerebellar right', 'SCR', 'right', 8, 'VBR', 'VB', 'Posterior', True),
    (29, 'inferior cerebellar left', 'ICL', 'left', 7, 'VBL', 'VB', 'Posterior', True),
    (30, 'inferior cerebellar right', 'ICR', 'right', 8, 'VBR', 'VB', 'Posterior', True),
    (31, 'lateral ventricle left', 'LVL', 'left', 9, 'LVL', 'LV', 'Ventricle', False),
    (32, 'lateral ventricle right', 'LVR', 'right', 10, 'LVR', 'LV', 'Ventricle', False),
]

schema_columns = [
    'fine_label',
    'fine_name',
    'fine_acronym',
    'hemisphere',
    'level2_label',
    'level2_acronym',
    'bilateral_family',
    'major_group',
    'use_as_graph_node',
]
schema_df = pd.DataFrame(rows, columns=schema_columns)

schema_df['source_lookup_note'] = ''
correction_notes = {
    10: 'Source repeats MCAPL for a right-sided parcel; canonical acronym is MCAPR.',
    12: 'Source repeats MCATL for a right-sided parcel; canonical acronym is MCATR.',
    14: 'Source repeats MCAOL for a right-sided parcel; canonical acronym is MCAOR.',
    16: 'Source repeats MCAIL for a right-sided parcel; canonical acronym is MCAIR.',
    17: 'Source text and intensity indicate left/level2=5; source level2 acronym PCAR is swapped. Canonical parent is PCAL.',
    18: 'Source text and intensity indicate right/level2=6; source level2 acronym PCAL is swapped. Canonical parent is PCAR.',
}
for label, note in correction_notes.items():
    schema_df.loc[schema_df['fine_label'] == label, 'source_lookup_note'] = note

fine_values = set(int(value) for value in np.unique(fine_labels))
expected_values = set(range(33))
if fine_values != expected_values:
    raise RuntimeError(
        f'Fine atlas labels differ from expected 0..32: {sorted(fine_values)}'
    )
if set(schema_df['fine_label']) != set(range(1, 33)):
    raise RuntimeError('Schema does not define every positive fine label exactly once.')
if schema_df['fine_label'].duplicated().any():
    raise RuntimeError('Schema contains duplicate fine labels.')

vascular_nodes = schema_df[schema_df['use_as_graph_node']]
excluded_nodes = schema_df[~schema_df['use_as_graph_node']]
if set(vascular_nodes['fine_label']) != set(range(1, 31)):
    raise RuntimeError('Graph-node labels must be exactly 1..30.')
if set(excluded_nodes['fine_label']) != {31, 32}:
    raise RuntimeError('Only ventricular labels 31 and 32 may be excluded.')

fine_to_level2 = dict(
    zip(schema_df['fine_label'].astype(int), schema_df['level2_label'].astype(int))
)
canonical_level2 = np.zeros_like(fine_labels, dtype=np.int16)
for fine_label, parent_label in fine_to_level2.items():
    canonical_level2[fine_labels == fine_label] = parent_label

vascular_only_fine = fine_labels.copy()
vascular_only_fine[np.isin(vascular_only_fine, [31, 32])] = 0

mismatch_mask = canonical_level2 != published_level2
mismatch_voxels = int(mismatch_mask.sum())

mismatch_rows = []
if mismatch_voxels:
    published_values = published_level2[mismatch_mask]
    canonical_values = canonical_level2[mismatch_mask]
    pairs, counts = np.unique(
        np.stack([published_values, canonical_values], axis=1),
        axis=0,
        return_counts=True,
    )
    for pair, count in zip(pairs, counts):
        mismatch_rows.append(
            {
                'published_level2_label': int(pair[0]),
                'canonical_level2_label': int(pair[1]),
                'voxels': int(count),
            }
        )

mismatch_df = pd.DataFrame(
    mismatch_rows,
    columns=['published_level2_label', 'canonical_level2_label', 'voxels'],
)

derived_header = fine_image.header.copy()
derived_header.set_data_dtype(np.int16)
derived_header.set_data_shape(fine_labels.shape)

canonical_image = nib.Nifti1Image(
    canonical_level2,
    fine_image.affine,
    header=derived_header,
)
canonical_image.set_sform(fine_image.affine, code=2)
canonical_image.set_qform(fine_image.affine, code=0)
nib.save(canonical_image, str(CANONICAL_LEVEL2_PATH))

vascular_image = nib.Nifti1Image(
    vascular_only_fine,
    fine_image.affine,
    header=derived_header.copy(),
)
vascular_image.set_sform(fine_image.affine, code=2)
vascular_image.set_qform(fine_image.affine, code=0)
nib.save(vascular_image, str(VASCULAR_ONLY_FINE_PATH))

schema_df.to_csv(SCHEMA_CSV_PATH, index=False)
mismatch_df.to_csv(MISMATCH_CSV_PATH, index=False)

canonical_check_image, canonical_check = load_label_volume(CANONICAL_LEVEL2_PATH)
vascular_check_image, vascular_check = load_label_volume(VASCULAR_ONLY_FINE_PATH)

schema_checks = {
    'all_fine_labels_defined_once': set(schema_df['fine_label']) == set(range(1, 33)),
    'vascular_graph_nodes_are_labels_1_to_30': set(vascular_nodes['fine_label']) == set(range(1, 31)),
    'only_ventricles_are_excluded': set(excluded_nodes['fine_label']) == {31, 32},
    'eight_unilateral_vascular_parents': set(vascular_nodes['level2_label']) == set(range(1, 9)),
    'four_bilateral_vascular_families': set(vascular_nodes['bilateral_family']) == {'ACA', 'MCA', 'PCA', 'VB'},
    'canonical_level2_labels_are_0_to_10': set(int(v) for v in np.unique(canonical_check)) == set(range(11)),
    'vascular_only_fine_labels_are_0_to_30': set(int(v) for v in np.unique(vascular_check)) == set(range(31)),
    'canonical_geometry_preserved': (
        canonical_check.shape == fine_labels.shape
        and np.allclose(canonical_check_image.affine, fine_image.affine, atol=1e-5, rtol=1e-6)
    ),
    'vascular_only_geometry_preserved': (
        vascular_check.shape == fine_labels.shape
        and np.allclose(vascular_check_image.affine, fine_image.affine, atol=1e-5, rtol=1e-6)
    ),
}
schema_gate_passed = bool(all(schema_checks.values()))

schema_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'schema_version': SCHEMA_VERSION,
    'atlas_git_commit': ATLAS_COMMIT,
    'source_lookup_path': str(LOOKUP_PATH),
    'source_lookup_sha256': file_sha256(LOOKUP_PATH),
    'source_fine_atlas_path': str(FINE_ATLAS_PATH),
    'source_fine_atlas_sha256': file_sha256(FINE_ATLAS_PATH),
    'source_level2_path': str(PUBLISHED_LEVEL2_PATH),
    'source_level2_sha256': file_sha256(PUBLISHED_LEVEL2_PATH),
    'source_files_modified': False,
    'fine_graph_node_labels': list(range(1, 31)),
    'excluded_nonvascular_labels': [31, 32],
    'unilateral_parent_labels': list(range(1, 9)),
    'excluded_nonvascular_parent_labels': [9, 10],
    'bilateral_vascular_families': ['ACA', 'MCA', 'PCA', 'VB'],
    'lookup_corrections': {str(key): value for key, value in correction_notes.items()},
    'published_vs_canonical_level2_mismatch_voxels': mismatch_voxels,
    'schema_csv': str(SCHEMA_CSV_PATH),
    'canonical_level2_path': str(CANONICAL_LEVEL2_PATH),
    'canonical_level2_sha256': file_sha256(CANONICAL_LEVEL2_PATH),
    'vascular_only_fine_path': str(VASCULAR_ONLY_FINE_PATH),
    'vascular_only_fine_sha256': file_sha256(VASCULAR_ONLY_FINE_PATH),
}
SCHEMA_JSON_PATH.write_text(json.dumps(schema_record, indent=2), encoding='utf-8')

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'schema_version': SCHEMA_VERSION,
    'gate_passed': schema_gate_passed,
    'checks': schema_checks,
    'schema_json_path': str(SCHEMA_JSON_PATH),
    'schema_json_sha256': file_sha256(SCHEMA_JSON_PATH),
    'registration_performed': False,
    'patient_data_modified': False,
}
SCHEMA_GATE_PATH.write_text(json.dumps(gate_record, indent=2), encoding='utf-8')

print('\nFrozen vascular label schema:')
display(schema_df)

print('\nPublished versus canonical level-2 mismatches:')
print(f'Total mismatch voxels: {mismatch_voxels}')
display(mismatch_df)

print('\nSchema checks:')
display(
    pd.DataFrame(
        [{'check': key, 'passed': value} for key, value in schema_checks.items()]
    )
)

if not schema_gate_passed:
    failed = [key for key, value in schema_checks.items() if not value]
    raise RuntimeError(
        'TRIAL8 ATLAS SCHEMA FREEZE FAILED. Do not begin registration. '
        f'Failed checks: {failed}'
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCULAR-ATLAS SCHEMA FREEZE PASSED')
print('=' * 80)
print(f'Schema version:          {SCHEMA_VERSION}')
print(f'Vascular graph nodes:    {len(vascular_nodes)}')
print(f'Unilateral parents:      {vascular_nodes["level2_label"].nunique()}')
print(f'Bilateral families:      {vascular_nodes["bilateral_family"].nunique()}')
print('Excluded labels:         31 (LVL), 32 (LVR)')
print(f'Canonical level-2 map:   {CANONICAL_LEVEL2_PATH}')
print(f'Vascular-only fine map:  {VASCULAR_ONLY_FINE_PATH}')
print(f'Schema record:           {SCHEMA_JSON_PATH}')
print(f'Schema gate:             {SCHEMA_GATE_PATH}')
print('\nNo patient registration or model training was performed.')
print('Stop here and check the schema and mismatch output.')


In [ ]:
import os
import sys
import json
import time
import signal
import subprocess
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
ENVIRONMENT_RECORD_PATH = (
    PROJECT_ROOT / 'configs' / 'trial8_registration_environment.json'
)

PINNED_ANTSPYX = '0.6.3'
PINNED_SCIPY = '1.16.3'
NUMPY_SPEC = 'numpy<2.4'


def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None


def run_clean_import_smoke_test():
    smoke_code = r'''
import json
import numpy as np
import scipy
from scipy import ndimage
import ants

fixed = ants.from_numpy(np.zeros((8, 8, 8), dtype=np.float32))
moving = ants.from_numpy(np.zeros((8, 8, 8), dtype=np.float32))

print(json.dumps({
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "antspyx": getattr(ants, "__version__", "unknown"),
    "ants_dimension": int(fixed.dimension),
    "ndimage_available": callable(ndimage.binary_fill_holes),
}))
'''
    result = subprocess.run(
        [sys.executable, '-c', smoke_code],
        text=True,
        capture_output=True,
    )

    if result.returncode != 0:
        return False, None, result.stderr.strip()

    output_lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not output_lines:
        return False, None, 'Clean import smoke test returned no output.'

    try:
        versions = json.loads(output_lines[-1])
    except json.JSONDecodeError as exc:
        return False, None, f'Could not parse smoke-test output: {exc}'

    version_ok = (
        versions.get('scipy') == PINNED_SCIPY
        and versions.get('antspyx') == PINNED_ANTSPYX
        and versions.get('ndimage_available') is True
        and versions.get('ants_dimension') == 3
    )

    if not version_ok:
        return False, versions, 'Installed versions do not match the Trial8 pins.'

    return True, versions, None


def install_registration_stack():
    dependency_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--upgrade',
        '--no-cache-dir',
        NUMPY_SPEC,
        'pandas',
        'nibabel',
        'pyyaml',
        'statsmodels',
        'webcolors',
        'matplotlib',
        'Pillow',
        'requests',
        'scikit-learn',
        'packaging',
    ]
    subprocess.check_call(dependency_command)

    
    
    pinned_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--force-reinstall',
        '--no-deps',
        '--no-cache-dir',
        f'scipy=={PINNED_SCIPY}',
        f'antspyx=={PINNED_ANTSPYX}',
    ]
    subprocess.check_call(pinned_command)


metadata_versions = {
    'antspyx': installed_version('antspyx'),
    'scipy': installed_version('scipy'),
    'numpy': installed_version('numpy'),
}

loaded_sensitive_roots = sorted({
    module_name.split('.')[0]
    for module_name in sys.modules
    if module_name.split('.')[0] in {'scipy', 'ants', 'statsmodels'}
})

smoke_passed, smoke_versions, smoke_error = run_clean_import_smoke_test()
installation_performed = False

if not smoke_passed:
    print('Trial8 registration environment needs repair.')
    print(f'Installed metadata before repair: {metadata_versions}')
    if smoke_versions is not None:
        print(f'Clean-process versions before repair: {smoke_versions}')
    if smoke_error:
        print(f'Clean-process error: {smoke_error}')

    install_registration_stack()
    installation_performed = True

    smoke_passed, smoke_versions, smoke_error = run_clean_import_smoke_test()
    if not smoke_passed:
        raise RuntimeError(
            'Trial8 registration environment repair failed. '
            f'Clean-process error: {smoke_error}'
        )

ENVIRONMENT_RECORD_PATH.parent.mkdir(parents=True, exist_ok=True)
environment_record = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version,
    'python_executable': sys.executable,
    'pinned_antspyx': PINNED_ANTSPYX,
    'pinned_scipy': PINNED_SCIPY,
    'numpy_specification': NUMPY_SPEC,
    'clean_process_versions': smoke_versions,
    'clean_import_smoke_test_passed': bool(smoke_passed),
    'installation_performed': bool(installation_performed),
}
ENVIRONMENT_RECORD_PATH.write_text(
    json.dumps(environment_record, indent=2),
    encoding='utf-8',
)

restart_required = installation_performed or bool(loaded_sensitive_roots)

if restart_required:
    print('\nTrial8 ANTs/SciPy files are now consistent on disk.')
    if loaded_sensitive_roots:
        print(f'Loaded modules requiring a clean restart: {loaded_sensitive_roots}')
    print(f'Verified clean-process versions: {smoke_versions}')
    print(f'Saved environment record: {ENVIRONMENT_RECORD_PATH}')
    print('Colab will restart now to clear stale imported modules.')
    print('After reconnecting, run this environment cell once more.')
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), signal.SIGKILL)

print('\n' + '=' * 80)
print('TRIAL8 REGISTRATION ENVIRONMENT READY')
print('=' * 80)
print(f'Python:   {sys.version.split()[0]}')
print(f'NumPy:    {smoke_versions["numpy"]}')
print(f'SciPy:    {smoke_versions["scipy"]}')
print(f'ANTsPyX:  {smoke_versions["antspyx"]}')
print(f'Record:   {ENVIRONMENT_RECORD_PATH}')
print('\nNext: run the standalone Trial8 registration preflight cell.')


In [ ]:
import os
import sys
import ast
import json
import time
import shutil
import hashlib
import subprocess
import tempfile
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path




ITK_THREADS = min(4, os.cpu_count() or 2)
os.environ['ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS'] = str(ITK_THREADS)


PINNED_ANTSPYX = '0.6.3'
PINNED_SCIPY = '1.16.3'
PREFLIGHT_CODE_REVISION = 'r2_case_audit_shape_source'


def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None


dependency_versions = {
    'antspyx': installed_version('antspyx'),
    'scipy': installed_version('scipy'),
}

dependency_errors = []
if dependency_versions['antspyx'] != PINNED_ANTSPYX:
    dependency_errors.append(
        f'antspyx must be {PINNED_ANTSPYX}; '
        f'found {dependency_versions["antspyx"]}'
    )
if dependency_versions['scipy'] != PINNED_SCIPY:
    dependency_errors.append(
        f'scipy must be {PINNED_SCIPY}; '
        f'found {dependency_versions["scipy"]}'
    )

if dependency_errors:
    raise RuntimeError(
        'Trial8 registration environment is not ready. Run the standalone '
        'Trial8 registration environment setup cell, allow its clean restart, '
        'then run that setup cell once more. Problems: '
        + '; '.join(dependency_errors)
    )

try:
    import numpy as np
    import pandas as pd
    import nibabel as nib
    import scipy
    from scipy import ndimage
    import matplotlib
    import matplotlib.pyplot as plt
    import ants
except Exception as exc:
    raise RuntimeError(
        'The Trial8 scientific stack could not be imported cleanly. '
        'Run the standalone registration environment setup cell and allow '
        'Colab to restart before retrying this preflight.'
    ) from exc

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')
print(f'Trial8 registration preflight revision: {PREFLIGHT_CODE_REVISION}')






PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SCHEMA_VERSION = 'vascular_hierarchy_v1'
PREFLIGHT_VERSION = 'v1_jhu_to_ncct_synra_mi'
SEED = 20240824

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
VERIFICATION_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'final_verification'
    / 'r1_verification_gate.json'
)
VERIFICATION_CASE_AUDIT_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'final_verification'
    / 'r1_case_audit.csv'
)

ATLAS_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
)
ACQUISITION_GATE_PATH = (
    ATLAS_ROOT / 'acquisition_audit' / 'atlas_acquisition_gate.json'
)
SCHEMA_DIR = ATLAS_ROOT / 'schema' / SCHEMA_VERSION
SCHEMA_GATE_PATH = SCHEMA_DIR / 'schema_freeze_gate.json'
SCHEMA_JSON_PATH = SCHEMA_DIR / 'vascular_label_schema.json'
SCHEMA_CSV_PATH = SCHEMA_DIR / 'vascular_label_schema.csv'
LABEL_AUDIT_PATH = (
    ATLAS_ROOT
    / 'label_hierarchy_audit'
    / 'atlas_label_hierarchy_audit.json'
)
JHU_REFERENCE_PATH = (
    ATLAS_ROOT
    / 'reference'
    / 'JHU_MNI_SS'
    / 'JHU_MNI_SS_T1_brain_182x218x182.nii.gz'
)
FULL_FINE_ATLAS_PATH = (
    ATLAS_ROOT
    / 'source'
    / 'Arterial_Atlas'
    / 'data'
    / 'Atlas_182'
    / 'ArterialAtlas.nii'
)

PREFLIGHT_ROOT = ATLAS_ROOT / 'registration_preflight' / PREFLIGHT_VERSION
CASE_ROOT = PREFLIGHT_ROOT / 'cases'
SELECTION_PATH = PREFLIGHT_ROOT / 'selected_train_cases.csv'
CONFIG_PATH = PREFLIGHT_ROOT / 'registration_preflight_config.json'
METRICS_PATH = PREFLIGHT_ROOT / 'registration_preflight_metrics.csv'
SUMMARY_PATH = PREFLIGHT_ROOT / 'registration_preflight_summary.json'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
SELECTION_QUANTILES = [0.10, 0.50, 0.90]
SELECTION_NAMES = ['small_crop', 'median_crop', 'large_crop']

REGISTRATION_CONFIG = {
    'type_of_transform': 'SyNRA',
    'aff_metric': 'mattes',
    'aff_sampling': 32,
    'aff_random_sampling_rate': 0.25,
    'syn_metric': 'mattes',
    'syn_sampling': 32,
    'reg_iterations': [80, 60, 40, 20],
    'grad_step': 0.15,
    'flow_sigma': 3,
    'total_sigma': 0,
    'mask_all_stages': True,
    'random_seed': SEED,
    'itk_threads': ITK_THREADS,
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


for path in [PREFLIGHT_ROOT, CASE_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()






required_paths = [
    FROZEN_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    VERIFICATION_CASE_AUDIT_PATH,
    ACQUISITION_GATE_PATH,
    SCHEMA_GATE_PATH,
    SCHEMA_JSON_PATH,
    SCHEMA_CSV_PATH,
    LABEL_AUDIT_PATH,
    JHU_REFERENCE_PATH,
    FULL_FINE_ATLAS_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required preflight inputs:\n' + '\n'.join(missing))

verification_gate = json.loads(VERIFICATION_GATE_PATH.read_text(encoding='utf-8'))
acquisition_gate = json.loads(ACQUISITION_GATE_PATH.read_text(encoding='utf-8'))
schema_gate = json.loads(SCHEMA_GATE_PATH.read_text(encoding='utf-8'))
schema_record = json.loads(SCHEMA_JSON_PATH.read_text(encoding='utf-8'))
label_audit_record = json.loads(LABEL_AUDIT_PATH.read_text(encoding='utf-8'))

if verification_gate.get('gate_passed') is not True:
    raise RuntimeError('Trial8 preprocessing verification gate is not passed.')
recorded_case_audit_path = Path(
    verification_gate.get('case_audit_path', '')
).resolve()
if recorded_case_audit_path != VERIFICATION_CASE_AUDIT_PATH.resolve():
    raise RuntimeError(
        'Verification gate points to an unexpected case-audit path.'
    )
if acquisition_gate.get('gate_passed') is not True:
    raise RuntimeError('Atlas acquisition gate is not passed.')
if schema_gate.get('gate_passed') is not True:
    raise RuntimeError('Atlas schema gate is not passed.')
if schema_record.get('schema_version') != SCHEMA_VERSION:
    raise RuntimeError('Unexpected atlas schema version.')
if file_sha256(SCHEMA_JSON_PATH) != schema_gate.get('schema_json_sha256'):
    raise RuntimeError('Frozen atlas schema changed after its gate was written.')
if file_sha256(FULL_FINE_ATLAS_PATH) != schema_record.get('source_fine_atlas_sha256'):
    raise RuntimeError('Fine atlas changed after schema freeze.')
if file_sha256(JHU_REFERENCE_PATH) != label_audit_record.get('jhu_reference_sha256'):
    raise RuntimeError('JHU reference changed after atlas schema freeze.')

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(f'Frozen manifest has {len(manifest_df)} rows; expected 148.')
split_counts = {
    str(key): int(value)
    for key, value in manifest_df['dev_split'].value_counts().to_dict().items()
}
if split_counts != EXPECTED_SPLITS:
    raise RuntimeError(f'Unexpected split counts: {split_counts}')
if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError('Frozen manifest contains duplicate subject IDs.')

required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen manifest is missing columns: {missing_columns}')

case_audit_df = pd.read_csv(VERIFICATION_CASE_AUDIT_PATH)
required_case_audit_columns = {'subject_id', 'valid', 'image_shape'}
missing_case_audit_columns = sorted(
    required_case_audit_columns - set(case_audit_df.columns)
)
if missing_case_audit_columns:
    raise RuntimeError(
        'Verification case audit is missing columns: '
        f'{missing_case_audit_columns}'
    )
if len(case_audit_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Verification case audit has {len(case_audit_df)} rows; expected 148.'
    )
if case_audit_df['subject_id'].duplicated().any():
    raise RuntimeError('Verification case audit contains duplicate subject IDs.')

case_audit_valid = case_audit_df['valid'].map(
    lambda value: value if isinstance(value, (bool, np.bool_))
    else str(value).strip().lower() == 'true'
)
if not case_audit_valid.all():
    invalid_subjects = case_audit_df.loc[
        ~case_audit_valid,
        'subject_id',
    ].astype(str).tolist()
    raise RuntimeError(
        f'Verification case audit contains invalid subjects: {invalid_subjects}'
    )

schema_df = pd.read_csv(SCHEMA_CSV_PATH)
graph_node_mask = schema_df['use_as_graph_node'].map(
    lambda value: value if isinstance(value, (bool, np.bool_))
    else str(value).strip().lower() == 'true'
)
vascular_schema_df = schema_df[graph_node_mask].copy()
if set(vascular_schema_df['fine_label'].astype(int)) != set(range(1, 31)):
    raise RuntimeError('Frozen schema does not define graph nodes 1..30.')

fine_to_level2 = dict(
    zip(schema_df['fine_label'].astype(int), schema_df['level2_label'].astype(int))
)

print('All Trial8 preprocessing and atlas gates passed.')
print(f'Manifest subjects: {len(manifest_df)}')
print(f'Split counts: {split_counts}')
print(f'ANTsPy version: {getattr(ants, "__version__", "unknown")}')
print(f'SciPy version: {scipy.__version__}')
print(f'ITK threads: {ITK_THREADS}')







def parse_shape(value):
    if isinstance(value, (tuple, list, np.ndarray)):
        result = tuple(int(v) for v in value)
    else:
        result = tuple(int(v) for v in ast.literal_eval(str(value)))
    if len(result) != 4:
        raise ValueError(f'Expected C x H x W x D shape, found {result}')
    return result


train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
train_df = train_df.merge(
    case_audit_df[['subject_id', 'image_shape']],
    on='subject_id',
    how='left',
    validate='one_to_one',
)
if train_df['image_shape'].isna().any():
    missing_shape_subjects = train_df.loc[
        train_df['image_shape'].isna(),
        'subject_id',
    ].astype(str).tolist()
    raise RuntimeError(
        'Audited image shape is missing for train subjects: '
        f'{missing_shape_subjects}'
    )

train_df['parsed_image_shape'] = train_df['image_shape'].map(parse_shape)
train_df['crop_voxels'] = train_df['parsed_image_shape'].map(
    lambda shape: int(np.prod(shape[1:], dtype=np.int64))
)
train_df = train_df.sort_values(['crop_voxels', 'subject_id']).reset_index(drop=True)

selected_rows = []
selected_subjects = set()
for quantile, selection_name in zip(SELECTION_QUANTILES, SELECTION_NAMES):
    target_volume = float(train_df['crop_voxels'].quantile(quantile))
    candidates = train_df[~train_df['subject_id'].isin(selected_subjects)].copy()
    candidates['distance_to_target'] = np.abs(candidates['crop_voxels'] - target_volume)
    row = candidates.sort_values(['distance_to_target', 'subject_id']).iloc[0]
    selected_subjects.add(row['subject_id'])
    selected_rows.append(
        {
            'selection_name': selection_name,
            'selection_quantile': quantile,
            'subject_id': row['subject_id'],
            'dev_split': row['dev_split'],
            'npz_path': row['vascufate_v2_npz_path'],
            'image_shape': str(row['parsed_image_shape']),
            'crop_voxels': int(row['crop_voxels']),
            'target_crop_voxels': target_volume,
        }
    )

selection_df = pd.DataFrame(selected_rows)
if set(selection_df['dev_split']) != {'train'}:
    raise RuntimeError('Registration preflight selection contains a non-train case.')
if selection_df['subject_id'].nunique() != len(SELECTION_QUANTILES):
    raise RuntimeError('Registration preflight subjects are not unique.')

selection_df.to_csv(SELECTION_PATH, index=False)

config_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
    'schema_version': SCHEMA_VERSION,
    'seed': SEED,
    'selection_method': 'train-only crop-volume quantiles; no target arrays read',
    'selection_shape_source': str(VERIFICATION_CASE_AUDIT_PATH),
    'selection_shape_source_sha256': file_sha256(
        VERIFICATION_CASE_AUDIT_PATH
    ),
    'selection_quantiles': SELECTION_QUANTILES,
    'registration': REGISTRATION_CONFIG,
    'fixed_image': '0.75 * ncct_brain_window + 0.25 * ncct_broad_window',
    'moving_image': str(JHU_REFERENCE_PATH),
    'label_source': str(FULL_FINE_ATLAS_PATH),
    'label_interpolator': 'genericLabel',
    'visual_qa_required': True,
    'bulk_registration_authorized': False,
    'validation_or_test_processed': False,
    'target_arrays_read': False,
}
CONFIG_PATH.write_text(json.dumps(config_record, indent=2), encoding='utf-8')

print('\nSelected train-only registration preflight cases:')
display(selection_df)







def save_nifti(data, affine, path, dtype=np.float32):
    array = np.asarray(data, dtype=dtype)
    image = nib.Nifti1Image(array, np.asarray(affine, dtype=np.float64))
    image.set_sform(np.asarray(affine, dtype=np.float64), code=2)
    image.set_qform(np.asarray(affine, dtype=np.float64), code=1)
    nib.save(image, str(path))


def dice_binary(first, second, eps=1e-6):
    first = np.asarray(first, dtype=bool)
    second = np.asarray(second, dtype=bool)
    return float(
        (2.0 * np.logical_and(first, second).sum() + eps)
        / (first.sum() + second.sum() + eps)
    )


def normalized_mutual_information(first, second, mask, bins=64):
    first = np.asarray(first, dtype=np.float32)
    second = np.asarray(second, dtype=np.float32)
    mask = np.asarray(mask, dtype=bool) & np.isfinite(first) & np.isfinite(second)
    if mask.sum() < 100:
        return float('nan')

    x = first[mask]
    y = second[mask]
    histogram, _, _ = np.histogram2d(x, y, bins=bins)
    probability = histogram / max(histogram.sum(), 1.0)
    px = probability.sum(axis=1)
    py = probability.sum(axis=0)
    nz_joint = probability > 0
    nz_x = px > 0
    nz_y = py > 0
    h_joint = -float(np.sum(probability[nz_joint] * np.log(probability[nz_joint])))
    h_x = -float(np.sum(px[nz_x] * np.log(px[nz_x])))
    h_y = -float(np.sum(py[nz_y] * np.log(py[nz_y])))
    return float((h_x + h_y) / h_joint) if h_joint > 0 else float('nan')


def center_of_mass_world(mask, affine):
    coordinates = np.argwhere(np.asarray(mask, dtype=bool))
    if len(coordinates) == 0:
        return np.full(3, np.nan, dtype=np.float64)
    center_voxel = coordinates.mean(axis=0)
    return np.asarray(nib.affines.apply_affine(affine, center_voxel), dtype=np.float64)


def label_boundary(labels):
    labels = np.asarray(labels)
    boundary = np.zeros(labels.shape, dtype=bool)
    for axis in range(3):
        left = [slice(None)] * 3
        right = [slice(None)] * 3
        left[axis] = slice(0, -1)
        right[axis] = slice(1, None)
        difference = labels[tuple(left)] != labels[tuple(right)]
        boundary[tuple(left)] |= difference
        boundary[tuple(right)] |= difference
    return boundary & (labels > 0)


def display_slice(array, axis, index):
    if axis == 0:
        image = array[index, :, :]
    elif axis == 1:
        image = array[:, index, :]
    else:
        image = array[:, :, index]
    return np.rot90(image)


def save_visual_qa(
    fixed_array,
    affine_warped,
    nonlinear_warped,
    warped_fine,
    cow_label,
    lvo_proximity,
    fixed_mask,
    output_path,
    subject_id,
):
    center = np.rint(np.argwhere(fixed_mask).mean(axis=0)).astype(int)
    axes = [2, 1, 0]
    axis_names = ['axial', 'coronal', 'sagittal']
    indices = [int(center[axis]) for axis in axes]
    boundaries = label_boundary(warped_fine)
    ventricle_mask = np.isin(warped_fine, [31, 32])
    vessel_mask = cow_label > 0
    lvo_mask = lvo_proximity >= 0.5

    fig, plot_axes = plt.subplots(3, 4, figsize=(18, 13))
    for row, (axis, axis_name, index) in enumerate(zip(axes, axis_names, indices)):
        fixed_slice = display_slice(fixed_array, axis, index)
        affine_slice = display_slice(affine_warped, axis, index)
        nonlinear_slice = display_slice(nonlinear_warped, axis, index)
        labels_slice = display_slice(warped_fine, axis, index)
        boundary_slice = display_slice(boundaries, axis, index)
        ventricle_slice = display_slice(ventricle_mask, axis, index)
        vessel_slice = display_slice(vessel_mask, axis, index)
        lvo_slice = display_slice(lvo_mask, axis, index)

        plot_axes[row, 0].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 0].set_title(f'{axis_name}: patient NCCT')

        plot_axes[row, 1].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 1].imshow(affine_slice, cmap='magma', alpha=0.40)
        plot_axes[row, 1].set_title('affine JHU overlay')

        plot_axes[row, 2].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 2].imshow(nonlinear_slice, cmap='magma', alpha=0.40)
        plot_axes[row, 2].set_title('SyN JHU overlay')

        plot_axes[row, 3].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 3].imshow(
            np.ma.masked_where(labels_slice == 0, labels_slice),
            cmap='turbo',
            alpha=0.28,
            vmin=1,
            vmax=32,
        )
        plot_axes[row, 3].imshow(
            np.ma.masked_where(~boundary_slice, boundary_slice),
            cmap='cool',
            alpha=0.85,
        )
        if ventricle_slice.any():
            plot_axes[row, 3].contour(
                ventricle_slice,
                levels=[0.5],
                colors='lime',
                linewidths=0.8,
            )
        if vessel_slice.any():
            plot_axes[row, 3].contour(
                vessel_slice,
                levels=[0.5],
                colors='magenta',
                linewidths=0.7,
            )
        if lvo_slice.any():
            plot_axes[row, 3].contour(lvo_slice, levels=[0.5], colors='yellow', linewidths=1.0)
        plot_axes[row, 3].set_title('territories + ventricle/CoW/LVO')

        for column in range(4):
            plot_axes[row, column].axis('off')

    fig.suptitle(
        f'{subject_id} registration QA\n'
        'cyan=territory boundaries, lime=ventricles, magenta=CoW, yellow=LVO',
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def copy_transform_list(transform_paths, destination, prefix):
    copied = []
    for index, source in enumerate(transform_paths):
        source = Path(source)
        suffix = ''.join(source.suffixes)
        if suffix == '':
            suffix = '.transform'
        target = destination / f'{prefix}_{index:02d}{suffix}'
        shutil.copy2(source, target)
        copied.append(target.name)
    return copied


def load_full_fine_labels_3d():
    image = nib.load(str(FULL_FINE_ATLAS_PATH))
    data = np.asanyarray(image.dataobj)
    if data.ndim == 4 and data.shape[-1] == 1:
        data = data[..., 0]
    if data.ndim != 3:
        raise RuntimeError(f'Unexpected fine-atlas shape: {data.shape}')
    return image, np.rint(data).astype(np.int16)


fine_source_image, fine_source_labels = load_full_fine_labels_3d()







def process_subject(selection_row):
    subject_id = str(selection_row['subject_id'])
    final_subject_dir = CASE_ROOT / subject_id
    final_complete_path = final_subject_dir / 'complete.json'

    if final_complete_path.exists():
        print(f'Reusing completed preflight case: {subject_id}')
        completed = json.loads(final_complete_path.read_text(encoding='utf-8'))
        if completed.get('preflight_version') != PREFLIGHT_VERSION:
            raise RuntimeError(
                f'{subject_id} was created by a different preflight version.'
            )
        if completed.get('dev_split') != 'train':
            raise RuntimeError(f'{subject_id} cached preflight is not train-only.')
        return completed
    if final_subject_dir.exists():
        raise RuntimeError(
            f'Incomplete preflight directory exists for {subject_id}: '
            f'{final_subject_dir}. Move it aside before rerunning.'
        )

    npz_path = Path(selection_row['npz_path'])
    if not npz_path.exists():
        raise FileNotFoundError(f'Missing Trial8 NPZ for {subject_id}: {npz_path}')

    local_parent = Path(tempfile.mkdtemp(prefix=f'trial8_reg_{subject_id}_', dir='/tmp'))
    local_case_dir = local_parent / subject_id
    local_case_dir.mkdir(parents=True, exist_ok=False)

    start_time = time.time()
    try:
        with np.load(npz_path, allow_pickle=False) as data:
            required_keys = {
                'image',
                'crop_mask',
                'cow_label',
                'lvo_proximity',
                'input_channels',
                'affine',
                'subject_id',
            }
            missing_keys = sorted(required_keys - set(data.files))
            if missing_keys:
                raise RuntimeError(f'{subject_id} NPZ missing keys: {missing_keys}')

            stored_subject_id = str(data['subject_id'].item())
            if stored_subject_id != subject_id:
                raise RuntimeError(
                    f'NPZ subject mismatch: expected {subject_id}, found {stored_subject_id}'
                )

            channel_names = [str(value) for value in data['input_channels'].tolist()]
            brain_index = channel_names.index('ncct_brain_window')
            broad_index = channel_names.index('ncct_broad_window')

            image_array = data['image']
            image_shape = tuple(int(value) for value in image_array.shape)
            brain_window = image_array[brain_index].astype(np.float32)
            broad_window = image_array[broad_index].astype(np.float32)
            fixed_mask_np = data['crop_mask'].astype(bool)
            cow_label = data['cow_label'].astype(np.uint8)
            lvo_proximity = data['lvo_proximity'].astype(np.float32)
            patient_affine = data['affine'].astype(np.float64)

        if image_shape[0] != len(channel_names):
            raise RuntimeError(f'{subject_id} channel metadata mismatch.')
        if brain_window.shape != fixed_mask_np.shape:
            raise RuntimeError(f'{subject_id} fixed image and mask shapes differ.')
        if ''.join(nib.aff2axcodes(patient_affine)) != 'RAS':
            raise RuntimeError(f'{subject_id} Trial8 crop is not RAS-oriented.')
        if not fixed_mask_np.any():
            raise RuntimeError(f'{subject_id} fixed registration mask is empty.')

        fixed_array = np.clip(
            0.75 * brain_window + 0.25 * broad_window,
            0.0,
            1.0,
        ).astype(np.float32)
        fixed_array[~fixed_mask_np] = 0.0

        fixed_path = local_case_dir / 'patient_ncct_fixed.nii.gz'
        fixed_mask_path = local_case_dir / 'patient_registration_mask.nii.gz'
        moving_path = local_case_dir / 'jhu_mni_ss_moving.nii.gz'
        moving_mask_path = local_case_dir / 'jhu_mni_ss_mask.nii.gz'
        full_fine_path = local_case_dir / 'arterial_atlas_full_fine_3d.nii.gz'

        save_nifti(fixed_array, patient_affine, fixed_path, np.float32)
        save_nifti(fixed_mask_np.astype(np.uint8), patient_affine, fixed_mask_path, np.uint8)
        shutil.copy2(JHU_REFERENCE_PATH, moving_path)

        jhu_image = nib.load(str(JHU_REFERENCE_PATH))
        jhu_data = np.asanyarray(jhu_image.dataobj).astype(np.float32)
        if jhu_data.ndim == 4 and jhu_data.shape[-1] == 1:
            jhu_data = jhu_data[..., 0]
        moving_mask_np = jhu_data > 0
        save_nifti(moving_mask_np.astype(np.uint8), jhu_image.affine, moving_mask_path, np.uint8)
        save_nifti(
            fine_source_labels,
            fine_source_image.affine,
            full_fine_path,
            np.int16,
        )

        fixed = ants.image_read(str(fixed_path), pixeltype='float')
        fixed_mask = ants.image_read(str(fixed_mask_path), pixeltype='unsigned char')
        moving = ants.image_read(str(moving_path), pixeltype='float')
        moving_mask = ants.image_read(str(moving_mask_path), pixeltype='unsigned char')
        moving_labels = ants.image_read(str(full_fine_path), pixeltype='float')

        print(f'\nRegistering {subject_id} with SyNRA + Mattes MI...')
        registration = ants.registration(
            fixed=fixed,
            moving=moving,
            type_of_transform=REGISTRATION_CONFIG['type_of_transform'],
            outprefix=str(local_case_dir / 'ants_'),
            mask=fixed_mask,
            moving_mask=moving_mask,
            mask_all_stages=REGISTRATION_CONFIG['mask_all_stages'],
            grad_step=REGISTRATION_CONFIG['grad_step'],
            flow_sigma=REGISTRATION_CONFIG['flow_sigma'],
            total_sigma=REGISTRATION_CONFIG['total_sigma'],
            aff_metric=REGISTRATION_CONFIG['aff_metric'],
            aff_sampling=REGISTRATION_CONFIG['aff_sampling'],
            aff_random_sampling_rate=REGISTRATION_CONFIG['aff_random_sampling_rate'],
            syn_metric=REGISTRATION_CONFIG['syn_metric'],
            syn_sampling=REGISTRATION_CONFIG['syn_sampling'],
            reg_iterations=tuple(REGISTRATION_CONFIG['reg_iterations']),
            random_seed=REGISTRATION_CONFIG['random_seed'],
            smoothing_in_mm=True,
            verbose=True,
        )

        forward_transforms = list(registration['fwdtransforms'])
        inverse_transforms = list(registration['invtransforms'])
        affine_transforms = [
            path for path in forward_transforms if str(path).lower().endswith('.mat')
        ]
        warp_transforms = [
            path
            for path in forward_transforms
            if str(path).lower().endswith(('.nii', '.nii.gz'))
            and 'inversewarp' not in str(path).lower()
        ]
        if not affine_transforms:
            raise RuntimeError(f'{subject_id} registration produced no affine transform.')
        if not warp_transforms:
            raise RuntimeError(f'{subject_id} registration produced no forward warp.')

        affine_warped = ants.apply_transforms(
            fixed=fixed,
            moving=moving,
            transformlist=affine_transforms,
            interpolator='linear',
        )
        nonlinear_warped = registration['warpedmovout']
        affine_mask = ants.apply_transforms(
            fixed=fixed,
            moving=moving_mask,
            transformlist=affine_transforms,
            interpolator='nearestNeighbor',
        )
        nonlinear_mask = ants.apply_transforms(
            fixed=fixed,
            moving=moving_mask,
            transformlist=forward_transforms,
            interpolator='nearestNeighbor',
        )
        warped_labels_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving_labels,
            transformlist=forward_transforms,
            interpolator='genericLabel',
        )

        affine_warped_np = affine_warped.numpy().astype(np.float32)
        nonlinear_warped_np = nonlinear_warped.numpy().astype(np.float32)
        affine_mask_np = affine_mask.numpy() > 0.5
        nonlinear_mask_np = nonlinear_mask.numpy() > 0.5
        warped_full_fine = np.rint(warped_labels_image.numpy()).astype(np.int16)

        unexpected_labels = sorted(
            set(int(value) for value in np.unique(warped_full_fine))
            - set(range(33))
        )
        warped_vascular_fine = warped_full_fine.copy()
        warped_vascular_fine[np.isin(warped_vascular_fine, [31, 32])] = 0

        warped_level2 = np.zeros_like(warped_full_fine, dtype=np.int16)
        for fine_label, parent_label in fine_to_level2.items():
            warped_level2[warped_full_fine == fine_label] = parent_label

        affine_output_path = local_case_dir / 'jhu_affine_warped.nii.gz'
        nonlinear_output_path = local_case_dir / 'jhu_syn_warped.nii.gz'
        fine_output_path = local_case_dir / 'arterial_atlas_full_fine_warped.nii.gz'
        vascular_output_path = local_case_dir / 'arterial_atlas_vascular30_warped.nii.gz'
        level2_output_path = local_case_dir / 'arterial_atlas_level2_derived_warped.nii.gz'
        jacobian_output_path = local_case_dir / 'syn_jacobian_determinant.nii.gz'
        qa_output_path = local_case_dir / 'registration_visual_qa.png'

        save_nifti(affine_warped_np, patient_affine, affine_output_path, np.float32)
        save_nifti(nonlinear_warped_np, patient_affine, nonlinear_output_path, np.float32)
        save_nifti(warped_full_fine, patient_affine, fine_output_path, np.int16)
        save_nifti(warped_vascular_fine, patient_affine, vascular_output_path, np.int16)
        save_nifti(warped_level2, patient_affine, level2_output_path, np.int16)

        jacobian_image = ants.create_jacobian_determinant_image(
            fixed,
            warp_transforms[0],
            do_log=False,
            geom=False,
        )
        jacobian_np = jacobian_image.numpy().astype(np.float32)
        save_nifti(jacobian_np, patient_affine, jacobian_output_path, np.float32)

        fixed_mask_count = int(fixed_mask_np.sum())
        atlas_support = warped_full_fine > 0
        atlas_support_count = int(atlas_support.sum())
        atlas_inside_count = int(np.logical_and(atlas_support, fixed_mask_np).sum())
        atlas_inside_fraction = (
            atlas_inside_count / atlas_support_count if atlas_support_count else 0.0
        )

        overlap_mask_affine = fixed_mask_np & affine_mask_np
        overlap_mask_nonlinear = fixed_mask_np & nonlinear_mask_np
        affine_nmi = normalized_mutual_information(
            fixed_array,
            affine_warped_np,
            overlap_mask_affine,
        )
        nonlinear_nmi = normalized_mutual_information(
            fixed_array,
            nonlinear_warped_np,
            overlap_mask_nonlinear,
        )

        fixed_center = center_of_mass_world(fixed_mask_np, patient_affine)
        affine_center = center_of_mass_world(affine_mask_np, patient_affine)
        nonlinear_center = center_of_mass_world(nonlinear_mask_np, patient_affine)

        jacobian_in_mask = jacobian_np[fixed_mask_np]
        jacobian_nonpositive_fraction = float(
            np.mean(jacobian_in_mask <= 0)
        )

        present_vascular_labels = sorted(
            int(value)
            for value in np.unique(warped_vascular_fine)
            if 1 <= int(value) <= 30
        )
        present_homolog_pairs = 0
        correctly_ordered_pairs = 0
        homolog_x_differences = []
        for left_label in range(1, 30, 2):
            right_label = left_label + 1
            left_mask = warped_full_fine == left_label
            right_mask = warped_full_fine == right_label
            if left_mask.any() and right_mask.any():
                present_homolog_pairs += 1
                left_x = center_of_mass_world(left_mask, patient_affine)[0]
                right_x = center_of_mass_world(right_mask, patient_affine)[0]
                difference = float(right_x - left_x)
                homolog_x_differences.append(difference)
                if difference > 0:
                    correctly_ordered_pairs += 1

        save_visual_qa(
            fixed_array=fixed_array,
            affine_warped=affine_warped_np,
            nonlinear_warped=nonlinear_warped_np,
            warped_fine=warped_full_fine,
            cow_label=cow_label,
            lvo_proximity=lvo_proximity,
            fixed_mask=fixed_mask_np,
            output_path=qa_output_path,
            subject_id=subject_id,
        )

        copied_forward = copy_transform_list(
            forward_transforms,
            local_case_dir,
            'forward_transform',
        )
        copied_inverse = copy_transform_list(
            inverse_transforms,
            local_case_dir,
            'inverse_transform',
        )

        elapsed_seconds = float(time.time() - start_time)
        metrics = {
            'preflight_version': PREFLIGHT_VERSION,
            'subject_id': subject_id,
            'selection_name': str(selection_row['selection_name']),
            'dev_split': 'train',
            'npz_path': str(npz_path),
            'fixed_shape': list(fixed_array.shape),
            'fixed_orientation': ''.join(nib.aff2axcodes(patient_affine)),
            'fixed_mask_voxels': fixed_mask_count,
            'affine_mask_dice': dice_binary(fixed_mask_np, affine_mask_np),
            'nonlinear_mask_dice': dice_binary(fixed_mask_np, nonlinear_mask_np),
            'affine_nmi': affine_nmi,
            'nonlinear_nmi': nonlinear_nmi,
            'affine_center_distance_mm': float(np.linalg.norm(fixed_center - affine_center)),
            'nonlinear_center_distance_mm': float(np.linalg.norm(fixed_center - nonlinear_center)),
            'atlas_support_voxels': atlas_support_count,
            'atlas_inside_fixed_mask_fraction': float(atlas_inside_fraction),
            'present_vascular_label_count': len(present_vascular_labels),
            'present_vascular_labels': present_vascular_labels,
            'ventricle_labels_present': [
                label for label in [31, 32] if np.any(warped_full_fine == label)
            ],
            'unexpected_labels': unexpected_labels,
            'present_homolog_pairs': present_homolog_pairs,
            'correctly_ordered_homolog_pairs': correctly_ordered_pairs,
            'minimum_right_minus_left_centroid_mm': float(
                min(homolog_x_differences)
            ) if homolog_x_differences else float('nan'),
            'jacobian_min': float(np.min(jacobian_in_mask)),
            'jacobian_p01': float(np.quantile(jacobian_in_mask, 0.01)),
            'jacobian_median': float(np.median(jacobian_in_mask)),
            'jacobian_p99': float(np.quantile(jacobian_in_mask, 0.99)),
            'jacobian_max': float(np.max(jacobian_in_mask)),
            'jacobian_nonpositive_fraction': jacobian_nonpositive_fraction,
            'forward_transform_files': copied_forward,
            'inverse_transform_files': copied_inverse,
            'visual_qa_file': qa_output_path.name,
            'elapsed_seconds': elapsed_seconds,
            'target_arrays_read': False,
        }

        safety_checks = {
            'train_subject_only': True,
            'fixed_orientation_is_ras': metrics['fixed_orientation'] == 'RAS',
            'no_unexpected_labels': len(unexpected_labels) == 0,
            'at_least_24_of_30_vascular_labels_present': len(present_vascular_labels) >= 24,
            'at_least_12_homolog_pairs_present': present_homolog_pairs >= 12,
            'all_present_homolog_pairs_have_correct_left_right_order': (
                correctly_ordered_pairs == present_homolog_pairs
            ),
            'no_nonpositive_jacobians_in_fixed_mask': (
                jacobian_nonpositive_fraction <= 1e-6
            ),
            'at_least_80_percent_atlas_support_inside_fixed_mask': (
                atlas_inside_fraction >= 0.80
            ),
        }
        metrics['automated_safety_checks'] = safety_checks
        metrics['automated_safety_passed'] = bool(all(safety_checks.values()))
        metrics['visual_qa_approved'] = False
        metrics['bulk_registration_authorized'] = False

        complete_path = local_case_dir / 'complete.json'
        complete_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

        shutil.copytree(local_case_dir, final_subject_dir)
        print(
            f'Completed {subject_id}: '
            f'safety={metrics["automated_safety_passed"]}, '
            f'labels={len(present_vascular_labels)}/30, '
            f'Jac<=0={jacobian_nonpositive_fraction:.6g}, '
            f'{elapsed_seconds / 60:.1f} min'
        )
        return metrics
    finally:
        if local_parent.exists():
            shutil.rmtree(local_parent)






metrics_rows = []
for index, row in selection_df.iterrows():
    print('\n' + '=' * 80)
    print(
        f'Registration preflight {index + 1}/{len(selection_df)}: '
        f'{row["subject_id"]} ({row["selection_name"]})'
    )
    print('=' * 80)
    metrics_rows.append(process_subject(row))

flat_metric_rows = []
for metrics in metrics_rows:
    flat_metric_rows.append(
        {
            key: value
            for key, value in metrics.items()
            if not isinstance(value, (dict, list))
        }
    )

metrics_df = pd.DataFrame(flat_metric_rows)
metrics_df.to_csv(METRICS_PATH, index=False)

all_train = all(row['dev_split'] == 'train' for row in metrics_rows)
all_automated_safety = all(
    bool(row['automated_safety_passed']) for row in metrics_rows
)
all_visual_files_exist = all(
    (CASE_ROOT / row['subject_id'] / row['visual_qa_file']).exists()
    for row in metrics_rows
)

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'subjects': [row['subject_id'] for row in metrics_rows],
    'all_subjects_are_train': all_train,
    'all_automated_safety_checks_passed': all_automated_safety,
    'all_visual_qa_files_exist': all_visual_files_exist,
    'visual_qa_approved': False,
    'bulk_registration_authorized': False,
    'validation_or_test_processed': False,
    'target_arrays_read': False,
    'metrics_path': str(METRICS_PATH),
    'case_root': str(CASE_ROOT),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('\nRegistration preflight metrics:')
display(metrics_df)

print('\n' + '=' * 80)
print('TRIAL8 TRAIN-ONLY REGISTRATION PREFLIGHT COMPLETE')
print('=' * 80)
print(f'All subjects train-only:         {all_train}')
print(f'Automated safety passed:         {all_automated_safety}')
print(f'Visual QA files created:         {all_visual_files_exist}')
print('Visual QA approved:              False')
print('Bulk registration authorized:    False')
print(f'Metrics:                         {METRICS_PATH}')
print(f'Case outputs:                    {CASE_ROOT}')
print(f'Summary:                         {SUMMARY_PATH}')
print('\nNo validation/test subjects or target arrays were used.')
print('Stop here and check every visual QA image before bulk registration.')

if not all_train or not all_visual_files_exist:
    raise RuntimeError(
        'Registration preflight output integrity failed. Do not continue.'
    )

if not all_automated_safety:
    raise RuntimeError(
        'One or more automated registration safety checks failed. '
        'Do not continue to bulk registration; check the saved metrics and QA.'
    )


In [ ]:
from pathlib import Path

import pandas as pd
from PIL import Image
from IPython.display import display, Markdown
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
PREFLIGHT_VERSION = 'v1_jhu_to_ncct_synra_mi'

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'registration_preflight'
    / PREFLIGHT_VERSION
)
METRICS_PATH = PREFLIGHT_ROOT / 'registration_preflight_metrics.csv'
CASE_ROOT = PREFLIGHT_ROOT / 'cases'

if not METRICS_PATH.exists():
    raise FileNotFoundError(f'Missing preflight metrics: {METRICS_PATH}')
if not CASE_ROOT.exists():
    raise FileNotFoundError(f'Missing preflight case directory: {CASE_ROOT}')

metrics_df = pd.read_csv(METRICS_PATH)

expected_subjects = {
    'sub-stroke0106',
    'sub-stroke0014',
    'sub-stroke0019',
}
observed_subjects = set(metrics_df['subject_id'].astype(str))
if observed_subjects != expected_subjects:
    raise RuntimeError(
        'Unexpected preflight subjects. '
        f'Expected {sorted(expected_subjects)}, found {sorted(observed_subjects)}.'
    )

required_columns = {
    'subject_id',
    'selection_name',
    'affine_mask_dice',
    'nonlinear_mask_dice',
    'affine_nmi',
    'nonlinear_nmi',
    'affine_center_distance_mm',
    'nonlinear_center_distance_mm',
    'atlas_inside_fixed_mask_fraction',
    'present_vascular_label_count',
    'present_homolog_pairs',
    'correctly_ordered_homolog_pairs',
    'jacobian_min',
    'jacobian_p01',
    'jacobian_median',
    'jacobian_p99',
    'jacobian_max',
    'jacobian_nonpositive_fraction',
    'automated_safety_passed',
    'target_arrays_read',
}
missing_columns = sorted(required_columns - set(metrics_df.columns))
if missing_columns:
    raise RuntimeError(f'Metrics are missing columns: {missing_columns}')

metrics_df['mask_dice_change_syn_minus_affine'] = (
    metrics_df['nonlinear_mask_dice'] - metrics_df['affine_mask_dice']
)
metrics_df['nmi_change_syn_minus_affine'] = (
    metrics_df['nonlinear_nmi'] - metrics_df['affine_nmi']
)
metrics_df['center_distance_change_syn_minus_affine_mm'] = (
    metrics_df['nonlinear_center_distance_mm']
    - metrics_df['affine_center_distance_mm']
)

review_columns = [
    'subject_id',
    'selection_name',
    'affine_mask_dice',
    'nonlinear_mask_dice',
    'mask_dice_change_syn_minus_affine',
    'affine_nmi',
    'nonlinear_nmi',
    'nmi_change_syn_minus_affine',
    'affine_center_distance_mm',
    'nonlinear_center_distance_mm',
    'center_distance_change_syn_minus_affine_mm',
    'atlas_inside_fixed_mask_fraction',
    'present_vascular_label_count',
    'present_homolog_pairs',
    'correctly_ordered_homolog_pairs',
    'jacobian_min',
    'jacobian_p01',
    'jacobian_median',
    'jacobian_p99',
    'jacobian_max',
    'jacobian_nonpositive_fraction',
    'automated_safety_passed',
    'target_arrays_read',
]

print('Trial8 registration preflight metric review:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    260,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(
        metrics_df[review_columns]
        .sort_values('selection_name')
        .reset_index(drop=True)
    )

checklist = [
    'Patient skull and brain outline agree with the SyN JHU overlay.',
    'Midline, ventricles, and major sulcal geometry remain plausible.',
    'Territory boundaries remain inside brain tissue without gross spill.',
    'Left and right territories are not exchanged.',
    'Posterior and infratentorial territories remain anatomically plausible.',
    'CoW and LVO contours are plausible relative to vascular territories.',
    'SyN does not introduce obvious local stretching, pinching, or folding.',
]

display(Markdown('## Visual review checklist'))
for item in checklist:
    print(f'[ ] {item}')

for row in metrics_df.sort_values('selection_name').itertuples(index=False):
    qa_path = CASE_ROOT / str(row.subject_id) / 'registration_visual_qa.png'
    if not qa_path.exists():
        raise FileNotFoundError(f'Missing visual QA image: {qa_path}')

    display(Markdown(
        f'## {row.subject_id}: {row.selection_name}\n\n'
        f'- Affine mask Dice: `{row.affine_mask_dice:.4f}`\n'
        f'- SyN mask Dice: `{row.nonlinear_mask_dice:.4f}`\n'
        f'- Affine NMI: `{row.affine_nmi:.4f}`\n'
        f'- SyN NMI: `{row.nonlinear_nmi:.4f}`\n'
        f'- Atlas support inside patient mask: '
        f'`{row.atlas_inside_fixed_mask_fraction:.4f}`\n'
        f'- Jacobian p01 / median / p99: '
        f'`{row.jacobian_p01:.4f} / {row.jacobian_median:.4f} / '
        f'{row.jacobian_p99:.4f}`'
    ))

    with Image.open(qa_path) as qa_image:
        display(qa_image.copy())

print('\n' + '=' * 80)
print('VISUAL QA REVIEW DISPLAY COMPLETE')
print('=' * 80)
print('No approval flags or bulk-registration authorization were changed.')
print('Check all three panels before proceeding.')


In [ ]:
import os
import sys
import json
import shutil
import hashlib
import subprocess
import urllib.request
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'

SYNTHSTRIP_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
)
SOURCE_DIR = SYNTHSTRIP_ROOT / 'source'
PROVENANCE_PATH = SOURCE_DIR / 'synthstrip_source_provenance.json'
SCRIPT_PATH = SOURCE_DIR / 'mri_synthstrip'
MODEL_PATH = SOURCE_DIR / 'synthstrip.1.pt'
REQUIREMENTS_PATH = SOURCE_DIR / 'gpu.requirements.txt'

VENV_ROOT = Path('/tmp/acf_net_synthstrip_py310')
VENV_PYTHON = VENV_ROOT / 'bin' / 'python'
VENV_READY_PATH = VENV_ROOT / 'trial8_synthstrip_environment.json'

UV_VERSION = '0.8.17'
PYTHON_VERSION = '3.10.18'
PYTORCH_INDEX = 'https://download.pytorch.org/whl/cu118'

FREESURFER_REPOSITORY = 'https://github.com/freesurfer/freesurfer.git'
FREESURFER_BRANCH = 'dev'
REQUIREMENTS_URL = (
    'https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/'
    'requirements/gpu.requirements.txt'
)
MODEL_URL = (
    'https://surfer.nmr.mgh.harvard.edu/docs/synthstrip/'
    'requirements/synthstrip.1.pt'
)
EXPECTED_MODEL_SHA256 = (
    '37417f802196186441aae3e7f385d94f8a98c64a88acaeaa2723af995c653e33'
)


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def atomic_download(url, destination):
    destination = Path(destination)
    temporary = destination.with_name(destination.name + '.download')
    if temporary.exists():
        temporary.unlink()
    urllib.request.urlretrieve(url, temporary)
    temporary.replace(destination)


def resolve_freesurfer_commit():
    output = subprocess.check_output(
        [
            'git',
            'ls-remote',
            FREESURFER_REPOSITORY,
            f'refs/heads/{FREESURFER_BRANCH}',
        ],
        text=True,
    ).strip()
    if not output:
        raise RuntimeError('Could not resolve the FreeSurfer development commit.')
    commit = output.split()[0]
    if len(commit) != 40:
        raise RuntimeError(f'Unexpected FreeSurfer commit: {commit}')
    return commit


def ensure_uv():
    try:
        installed = metadata.version('uv')
    except metadata.PackageNotFoundError:
        installed = None

    if installed != UV_VERSION:
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--upgrade',
            f'uv=={UV_VERSION}',
        ])

    uv_path = shutil.which('uv')
    if uv_path is None:
        raise RuntimeError('uv installed but its executable was not found.')
    return Path(uv_path)


SOURCE_DIR.mkdir(parents=True, exist_ok=True)

if PROVENANCE_PATH.exists():
    provenance = json.loads(PROVENANCE_PATH.read_text(encoding='utf-8'))
    freesurfer_commit = str(provenance['freesurfer_commit'])
else:
    freesurfer_commit = resolve_freesurfer_commit()
    provenance = None

SCRIPT_URL = (
    'https://raw.githubusercontent.com/freesurfer/freesurfer/'
    f'{freesurfer_commit}/mri_synthstrip/mri_synthstrip'
)

if not SCRIPT_PATH.exists():
    atomic_download(SCRIPT_URL, SCRIPT_PATH)
if not MODEL_PATH.exists():
    atomic_download(MODEL_URL, MODEL_PATH)
if not REQUIREMENTS_PATH.exists():
    atomic_download(REQUIREMENTS_URL, REQUIREMENTS_PATH)

script_sha256 = file_sha256(SCRIPT_PATH)
model_sha256 = file_sha256(MODEL_PATH)
requirements_sha256 = file_sha256(REQUIREMENTS_PATH)

if model_sha256 != EXPECTED_MODEL_SHA256:
    raise RuntimeError(
        'SynthStrip model hash mismatch. '
        f'Expected {EXPECTED_MODEL_SHA256}, found {model_sha256}.'
    )

if provenance is not None:
    expected_hashes = {
        'script_sha256': script_sha256,
        'model_sha256': model_sha256,
        'requirements_sha256': requirements_sha256,
    }
    for key, observed in expected_hashes.items():
        if provenance.get(key) != observed:
            raise RuntimeError(
                f'Existing SynthStrip source changed: {key} '
                f'expected {provenance.get(key)}, found {observed}.'
            )
else:
    provenance = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'synthstrip_version': SYNTHSTRIP_VERSION,
        'freesurfer_repository': FREESURFER_REPOSITORY,
        'freesurfer_branch': FREESURFER_BRANCH,
        'freesurfer_commit': freesurfer_commit,
        'script_url': SCRIPT_URL,
        'script_sha256': script_sha256,
        'model_url': MODEL_URL,
        'model_sha256': model_sha256,
        'model_license_choice': 'MIT',
        'requirements_url': REQUIREMENTS_URL,
        'requirements_sha256': requirements_sha256,
        'official_python_version': PYTHON_VERSION,
        'official_gpu_requirements': REQUIREMENTS_PATH.read_text(encoding='utf-8'),
        'no_csf': False,
        'border_mm': 1,
    }
    PROVENANCE_PATH.write_text(
        json.dumps(provenance, indent=2),
        encoding='utf-8',
    )

uv_path = ensure_uv()

environment_needs_build = not VENV_READY_PATH.exists()
if environment_needs_build:
    if VENV_ROOT.exists():
        shutil.rmtree(VENV_ROOT)

    subprocess.check_call([
        str(uv_path),
        'venv',
        '--python',
        PYTHON_VERSION,
        '--seed',
        str(VENV_ROOT),
    ])

    subprocess.check_call([
        str(VENV_PYTHON),
        '-m',
        'pip',
        'install',
        '--quiet',
        '--no-cache-dir',
        '--extra-index-url',
        PYTORCH_INDEX,
        '-r',
        str(REQUIREMENTS_PATH),
    ])

gpu_smoke_code = r'''
import json
import torch
import numpy
import scipy
import nibabel
import surfa

record = {
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "nibabel": nibabel.__version__,
}
print(json.dumps(record))
if not torch.cuda.is_available():
    raise SystemExit("CUDA is not available inside the SynthStrip environment.")
'''

gpu_result = subprocess.run(
    [str(VENV_PYTHON), '-c', gpu_smoke_code],
    text=True,
    capture_output=True,
)
if gpu_result.returncode != 0:
    raise RuntimeError(
        'SynthStrip GPU environment smoke test failed:\n'
        + gpu_result.stdout
        + '\n'
        + gpu_result.stderr
    )

gpu_lines = [line for line in gpu_result.stdout.splitlines() if line.strip()]
gpu_record = json.loads(gpu_lines[-1])

help_result = subprocess.run(
    [str(VENV_PYTHON), str(SCRIPT_PATH), '--help'],
    text=True,
    capture_output=True,
)
help_text = help_result.stdout + '\n' + help_result.stderr
required_options = ['-i', '-o', '-m', '-b', '-g', '--model']
missing_options = [option for option in required_options if option not in help_text]

if help_result.returncode not in {0, 1} or missing_options:
    raise RuntimeError(
        'SynthStrip command-line smoke test failed. '
        f'Missing options: {missing_options}\n{help_text}'
    )

environment_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'venv_root': str(VENV_ROOT),
    'python_version': PYTHON_VERSION,
    'uv_version': UV_VERSION,
    'source_provenance_path': str(PROVENANCE_PATH),
    'source_provenance_sha256': file_sha256(PROVENANCE_PATH),
    'script_path': str(SCRIPT_PATH),
    'model_path': str(MODEL_PATH),
    'gpu_smoke_test': gpu_record,
    'cli_required_options': required_options,
    'cli_smoke_test_passed': True,
}
VENV_READY_PATH.write_text(
    json.dumps(environment_record, indent=2),
    encoding='utf-8',
)

print('\n' + '=' * 80)
print('TRIAL8 SYNTHSTRIP GPU ENVIRONMENT READY')
print('=' * 80)
print(f'Compute:              GPU')
print(f'GPU:                  {gpu_record["gpu_name"]}')
print(f'Isolated Python:      {PYTHON_VERSION}')
print(f'Torch:                {gpu_record["torch"]}')
print(f'Torch CUDA build:     {gpu_record["torch_cuda_build"]}')
print(f'NumPy:                {gpu_record["numpy"]}')
print(f'SciPy:                {gpu_record["scipy"]}')
print(f'FreeSurfer commit:    {freesurfer_commit}')
print(f'Model SHA-256:        {model_sha256}')
print('Study trainable parameters: 0 (preprocessing only)')
print(f'Environment:          {VENV_ROOT}')
print(f'Provenance:           {PROVENANCE_PATH}')
print('\nNext: run the standalone three-case SynthStrip mask preflight cell.')


In [ ]:
import json
import time
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
PREFLIGHT_VERSION = 'v1_jhu_to_ncct_synra_mi'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'

REGISTRATION_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'registration_preflight'
    / PREFLIGHT_VERSION
)
SELECTION_PATH = REGISTRATION_PREFLIGHT_ROOT / 'selected_train_cases.csv'

SYNTHSTRIP_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
)
SOURCE_DIR = SYNTHSTRIP_ROOT / 'source'
PROVENANCE_PATH = SOURCE_DIR / 'synthstrip_source_provenance.json'
SCRIPT_PATH = SOURCE_DIR / 'mri_synthstrip'
MODEL_PATH = SOURCE_DIR / 'synthstrip.1.pt'
PREFLIGHT_ROOT = SYNTHSTRIP_ROOT / 'preflight_train3'
CASE_ROOT = PREFLIGHT_ROOT / 'cases'
AUDIT_PATH = PREFLIGHT_ROOT / 'synthstrip_mask_preflight_audit.csv'
SUMMARY_PATH = PREFLIGHT_ROOT / 'synthstrip_mask_preflight_summary.json'

VENV_ROOT = Path('/tmp/acf_net_synthstrip_py310')
VENV_PYTHON = VENV_ROOT / 'bin' / 'python'
VENV_READY_PATH = VENV_ROOT / 'trial8_synthstrip_environment.json'

EXPECTED_SUBJECTS = {
    'sub-stroke0106',
    'sub-stroke0014',
    'sub-stroke0019',
}
EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)
BORDER_MM = 1


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def save_nifti(array, affine, path, dtype):
    array = np.asarray(array, dtype=dtype)
    image = nib.Nifti1Image(array, np.asarray(affine, dtype=np.float64))
    image.set_sform(np.asarray(affine, dtype=np.float64), code=2)
    image.set_qform(np.asarray(affine, dtype=np.float64), code=1)
    nib.save(image, str(path))


def display_slice(array, axis, index):
    return np.rot90(np.take(array, int(index), axis=axis))


def mask_boundary(mask):
    mask = np.asarray(mask, dtype=bool)
    eroded = ndimage.binary_erosion(mask, iterations=1)
    return mask & ~eroded


def save_mask_qa(brain_window, broad_window, mask, output_path, subject_id):
    coordinates = np.argwhere(mask)
    if len(coordinates):
        center = np.rint(coordinates.mean(axis=0)).astype(int)
    else:
        center = np.asarray(mask.shape) // 2
    boundary = mask_boundary(mask)
    axes = [2, 1, 0]
    names = ['axial', 'coronal', 'sagittal']

    fig, panels = plt.subplots(3, 3, figsize=(14, 13))
    for row, (axis, name) in enumerate(zip(axes, names)):
        index = int(center[axis])
        brain_slice = display_slice(brain_window, axis, index)
        broad_slice = display_slice(broad_window, axis, index)
        mask_slice = display_slice(mask, axis, index)
        boundary_slice = display_slice(boundary, axis, index)

        panels[row, 0].imshow(brain_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 0].set_title(f'{name}: NCCT brain window')

        panels[row, 1].imshow(broad_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 1].imshow(
            np.ma.masked_where(~mask_slice, mask_slice),
            cmap='Blues',
            alpha=0.28,
        )
        panels[row, 1].set_title('broad window + mask fill')

        panels[row, 2].imshow(brain_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 2].imshow(
            np.ma.masked_where(~boundary_slice, boundary_slice),
            cmap='autumn',
            alpha=0.95,
        )
        panels[row, 2].set_title('brain window + mask boundary')

        for column in range(3):
            panels[row, column].axis('off')

    fig.suptitle(
        f'{subject_id} SynthStrip mask QA\n'
        'default model with CSF, border=1 mm',
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


required_paths = [
    SELECTION_PATH,
    PROVENANCE_PATH,
    SCRIPT_PATH,
    MODEL_PATH,
    VENV_PYTHON,
    VENV_READY_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'SynthStrip setup is incomplete. Run the standalone environment setup '
        'cell first. Missing:\n' + '\n'.join(missing_paths)
    )

provenance = json.loads(PROVENANCE_PATH.read_text(encoding='utf-8'))
observed_script_sha256 = file_sha256(SCRIPT_PATH)
observed_model_sha256 = file_sha256(MODEL_PATH)
if provenance.get('script_sha256') != observed_script_sha256:
    raise RuntimeError('SynthStrip script does not match its frozen provenance.')
if provenance.get('model_sha256') != observed_model_sha256:
    raise RuntimeError('SynthStrip model does not match its frozen provenance.')

gpu_result = subprocess.run(
    [
        str(VENV_PYTHON),
        '-c',
        'import torch; print(torch.cuda.get_device_name(0)); '
        'raise SystemExit(0 if torch.cuda.is_available() else 1)',
    ],
    text=True,
    capture_output=True,
)
if gpu_result.returncode != 0:
    raise RuntimeError('CUDA is unavailable in the isolated SynthStrip environment.')

for path in [PREFLIGHT_ROOT, CASE_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

selection_df = pd.read_csv(SELECTION_PATH)
observed_subjects = set(selection_df['subject_id'].astype(str))
if observed_subjects != EXPECTED_SUBJECTS or len(selection_df) != 3:
    raise RuntimeError(
        f'Unexpected registration preflight selection: {sorted(observed_subjects)}'
    )
if set(selection_df['dev_split'].astype(str)) != {'train'}:
    raise RuntimeError('SynthStrip preflight selection contains a non-train case.')

print('Compute: GPU')
print(f'GPU: {gpu_result.stdout.strip()}')
print('Study trainable parameters: 0 (preprocessing only)')
print(f'SynthStrip source SHA-256: {observed_script_sha256}')
print(f'SynthStrip model SHA-256:  {observed_model_sha256}')
print('\nSelected train-only mask preflight cases:')
display(selection_df)

audit_rows = []
script_sha256 = observed_script_sha256
model_sha256 = observed_model_sha256

for case_index, row in enumerate(selection_df.itertuples(index=False), start=1):
    subject_id = str(row.subject_id)
    npz_path = Path(row.npz_path)
    case_dir = CASE_ROOT / subject_id
    case_dir.mkdir(parents=True, exist_ok=True)

    input_path = case_dir / 'ncct_broad_window_input.nii.gz'
    stripped_path = case_dir / 'ncct_synthstrip_brain.nii.gz'
    mask_path = case_dir / 'ncct_synthstrip_mask.nii.gz'
    qa_path = case_dir / 'synthstrip_mask_visual_qa.png'
    complete_path = case_dir / 'complete.json'

    print('\n' + '=' * 80)
    print(f'SynthStrip preflight {case_index}/3: {subject_id}')
    print('=' * 80)

    npz_sha256 = file_sha256(npz_path)
    reusable_outputs = [stripped_path, mask_path, qa_path, complete_path]
    if all(path.exists() for path in reusable_outputs):
        existing_record = json.loads(complete_path.read_text(encoding='utf-8'))
        reusable = (
            existing_record.get('subject_id') == subject_id
            and existing_record.get('npz_sha256') == npz_sha256
            and existing_record.get('script_sha256') == script_sha256
            and existing_record.get('model_sha256') == model_sha256
            and existing_record.get('mask_sha256') == file_sha256(mask_path)
            and existing_record.get('brain_sha256') == file_sha256(stripped_path)
            and existing_record.get('qa_sha256') == file_sha256(qa_path)
            and existing_record.get('target_arrays_read') is False
        )
        if reusable:
            audit_rows.append({
                key: value
                for key, value in existing_record.items()
                if not isinstance(value, (dict, list))
            })
            print(
                f'Reused completed {subject_id}: '
                f'safety={existing_record["automated_safety_passed"]}'
            )
            continue

        print('Existing outputs did not pass resume checks. Recomputing this case.')

    with np.load(npz_path, allow_pickle=False) as data:
        required_keys = {
            'image',
            'input_channels',
            'crop_mask',
            'cow_label',
            'lvo_proximity',
            'affine',
            'subject_id',
        }
        missing_keys = sorted(required_keys - set(data.files))
        if missing_keys:
            raise RuntimeError(f'{subject_id} NPZ missing keys: {missing_keys}')

        if str(data['subject_id'].item()) != subject_id:
            raise RuntimeError(f'{subject_id} NPZ subject identifier mismatch.')

        channel_names = [str(value) for value in data['input_channels'].tolist()]
        brain_index = channel_names.index('ncct_brain_window')
        broad_index = channel_names.index('ncct_broad_window')

        image = data['image']
        brain_window = image[brain_index].astype(np.float32)
        broad_window = image[broad_index].astype(np.float32)
        crop_mask = data['crop_mask'].astype(bool)
        cow_mask = data['cow_label'] > 0
        lvo_mask = data['lvo_proximity'] >= 0.5
        affine = data['affine'].astype(np.float64)

    spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
    orientation = ''.join(nib.aff2axcodes(affine))
    if orientation != 'RAS':
        raise RuntimeError(f'{subject_id} orientation is {orientation}, not RAS.')
    if not np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0):
        raise RuntimeError(f'{subject_id} spacing is {spacing.tolist()}.')

    save_nifti(broad_window, affine, input_path, np.float32)

    start_time = time.time()
    command = [
        str(VENV_PYTHON),
        str(SCRIPT_PATH),
        '-i',
        str(input_path),
        '-o',
        str(stripped_path),
        '-m',
        str(mask_path),
        '-b',
        str(BORDER_MM),
        '-g',
        '--model',
        str(MODEL_PATH),
    ]

    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            f'{subject_id} SynthStrip failed.\nSTDOUT:\n{result.stdout}\n'
            f'STDERR:\n{result.stderr}'
        )
    elapsed_seconds = float(time.time() - start_time)

    mask_image = nib.load(str(mask_path))
    mask_data = np.asanyarray(mask_image.dataobj)
    if mask_data.ndim == 4 and mask_data.shape[-1] == 1:
        mask_data = mask_data[..., 0]
    brain_mask = mask_data > 0.5

    if brain_mask.shape != broad_window.shape:
        raise RuntimeError(
            f'{subject_id} mask shape {brain_mask.shape} does not match '
            f'{broad_window.shape}.'
        )
    if not np.allclose(mask_image.affine, affine, atol=1e-5, rtol=1e-5):
        raise RuntimeError(f'{subject_id} mask affine does not match the NPZ grid.')

    labeled, component_count = ndimage.label(brain_mask)
    component_sizes = np.bincount(labeled.ravel())[1:]
    largest_component_fraction = (
        float(component_sizes.max() / brain_mask.sum())
        if len(component_sizes) and brain_mask.any()
        else 0.0
    )

    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    brain_volume_ml = float(brain_mask.sum() * voxel_volume_ml)
    crop_containment = float(
        np.logical_and(brain_mask, crop_mask).sum() / max(brain_mask.sum(), 1)
    )
    cow_coverage = float(
        np.logical_and(cow_mask, brain_mask).sum() / max(cow_mask.sum(), 1)
    ) if cow_mask.any() else 1.0
    lvo_coverage = float(
        np.logical_and(lvo_mask, brain_mask).sum() / max(lvo_mask.sum(), 1)
    ) if lvo_mask.any() else 1.0

    edge_touch_axes = {
        'x': bool(brain_mask[0].any() or brain_mask[-1].any()),
        'y': bool(brain_mask[:, 0].any() or brain_mask[:, -1].any()),
        'z': bool(brain_mask[:, :, 0].any() or brain_mask[:, :, -1].any()),
    }

    safety_checks = {
        'mask_nonempty': bool(brain_mask.any()),
        'brain_volume_between_500_and_2200_ml': 500.0 <= brain_volume_ml <= 2200.0,
        'largest_component_fraction_at_least_0p995': (
            largest_component_fraction >= 0.995
        ),
        'brain_mask_at_least_0p99_inside_crop_mask': crop_containment >= 0.99,
        'cow_coverage_at_least_0p95': cow_coverage >= 0.95,
        'lvo_coverage_at_least_0p95': lvo_coverage >= 0.95,
        'does_not_touch_x_or_y_crop_edges': not (
            edge_touch_axes['x'] or edge_touch_axes['y']
        ),
    }

    save_mask_qa(
        brain_window,
        broad_window,
        brain_mask,
        qa_path,
        subject_id,
    )

    record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'subject_id': subject_id,
        'selection_name': str(row.selection_name),
        'dev_split': str(row.dev_split),
        'npz_path': str(npz_path),
        'npz_sha256': npz_sha256,
        'input_path': str(input_path),
        'brain_path': str(stripped_path),
        'mask_path': str(mask_path),
        'qa_path': str(qa_path),
        'script_sha256': script_sha256,
        'model_sha256': model_sha256,
        'orientation': orientation,
        'spacing_mm': spacing.tolist(),
        'shape': list(brain_mask.shape),
        'brain_voxels': int(brain_mask.sum()),
        'brain_volume_ml': brain_volume_ml,
        'component_count': int(component_count),
        'largest_component_fraction': largest_component_fraction,
        'crop_containment_fraction': crop_containment,
        'cow_coverage_fraction': cow_coverage,
        'lvo_coverage_fraction': lvo_coverage,
        'edge_touch_axes': edge_touch_axes,
        'elapsed_seconds': elapsed_seconds,
        'compute': 'GPU',
        'no_csf': False,
        'border_mm': BORDER_MM,
        'target_arrays_read': False,
        'safety_checks': safety_checks,
        'automated_safety_passed': bool(all(safety_checks.values())),
        'visual_qa_approved': False,
        'registration_rerun_authorized': False,
        'bulk_processing_authorized': False,
    }
    record['brain_sha256'] = file_sha256(stripped_path)
    record['mask_sha256'] = file_sha256(mask_path)
    record['qa_sha256'] = file_sha256(qa_path)

    complete_temporary = complete_path.with_suffix('.json.tmp')
    complete_temporary.write_text(json.dumps(record, indent=2), encoding='utf-8')
    complete_temporary.replace(complete_path)

    audit_rows.append({
        key: value
        for key, value in record.items()
        if not isinstance(value, (dict, list))
    })

    pd.DataFrame(audit_rows).to_csv(AUDIT_PATH, index=False)

    print(
        f'Completed {subject_id}: volume={brain_volume_ml:.1f} mL, '
        f'CoW coverage={cow_coverage:.4f}, '
        f'LVO coverage={lvo_coverage:.4f}, '
        f'safety={record["automated_safety_passed"]}, '
        f'{elapsed_seconds:.1f} s'
    )

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(AUDIT_PATH, index=False)

all_safety_passed = bool(audit_df['automated_safety_passed'].all())
summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'synthstrip_version': SYNTHSTRIP_VERSION,
    'subjects': int(len(audit_df)),
    'all_train_only': bool(set(audit_df['dev_split']) == {'train'}),
    'all_automated_safety_passed': all_safety_passed,
    'all_visual_qa_approved': False,
    'target_arrays_read': False,
    'registration_rerun_authorized': False,
    'bulk_processing_authorized': False,
    'source_provenance_path': str(PROVENANCE_PATH),
    'audit_path': str(AUDIT_PATH),
    'case_root': str(CASE_ROOT),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('\nSynthStrip mask preflight audit:')
with pd.option_context('display.max_columns', None, 'display.width', 240):
    display(audit_df)

print('\n' + '=' * 80)
print('TRIAL8 THREE-CASE SYNTHSTRIP MASK PREFLIGHT COMPLETE')
print('=' * 80)
print(f'All train-only:                  {summary["all_train_only"]}')
print(f'Automated safety passed:         {all_safety_passed}')
print('Visual QA approved:              False')
print('Registration rerun authorized:  False')
print('Bulk processing authorized:      False')
print(f'Audit:                            {AUDIT_PATH}')
print(f'Case outputs:                     {CASE_ROOT}')
print(f'Summary:                          {SUMMARY_PATH}')
print('\nNo target arrays were read and no registration was rerun.')
print('Stop here and check every SynthStrip mask QA image.')


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / 'preflight_train3'
)
AUDIT_PATH = PREFLIGHT_ROOT / 'synthstrip_mask_preflight_audit.csv'

if not AUDIT_PATH.exists():
    raise FileNotFoundError(f'Missing SynthStrip preflight audit: {AUDIT_PATH}')

audit_df = pd.read_csv(AUDIT_PATH)
if len(audit_df) != 3:
    raise RuntimeError(f'Expected three mask-preflight cases, found {len(audit_df)}.')

review_columns = [
    'subject_id',
    'selection_name',
    'brain_volume_ml',
    'component_count',
    'largest_component_fraction',
    'crop_containment_fraction',
    'cow_coverage_fraction',
    'lvo_coverage_fraction',
    'elapsed_seconds',
    'compute',
    'automated_safety_passed',
    'target_arrays_read',
]
missing_columns = sorted(set(review_columns) - set(audit_df.columns))
if missing_columns:
    raise RuntimeError(f'Audit is missing columns: {missing_columns}')

print('Trial8 SynthStrip mask preflight metrics:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    220,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(audit_df[review_columns].sort_values('selection_name'))

display(Markdown('## Visual mask checklist'))
checklist = [
    'The mask includes both cerebral hemispheres and the cerebellum.',
    'The mask follows the intracranial boundary without including skull.',
    'No frontal, temporal, occipital, or superior cortex is truncated.',
    'The brainstem is included without extending far into the neck.',
    'Ventricular and surrounding CSF spaces remain within the default mask.',
    'There are no disconnected extracranial mask islands.',
]
for item in checklist:
    print(f'[ ] {item}')

for row in audit_df.sort_values('selection_name').itertuples(index=False):
    qa_path = Path(row.qa_path)
    if not qa_path.exists():
        raise FileNotFoundError(f'Missing mask QA image: {qa_path}')

    display(Markdown(
        f'## {row.subject_id}: {row.selection_name}\n\n'
        f'- Brain volume: `{row.brain_volume_ml:.1f} mL`\n'
        f'- Largest-component fraction: '
        f'`{row.largest_component_fraction:.6f}`\n'
        f'- CoW coverage: `{row.cow_coverage_fraction:.6f}`\n'
        f'- LVO coverage: `{row.lvo_coverage_fraction:.6f}`\n'
        f'- Automated safety: `{row.automated_safety_passed}`'
    ))

    with Image.open(qa_path) as image:
        display(image.copy())

print('\n' + '=' * 80)
print('TRIAL8 SYNTHSTRIP MASK VISUAL QA DISPLAY COMPLETE')
print('=' * 80)
print('No approval or authorization state was changed.')
print('Check every mask before revising registration.')


In [ ]:
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
APPROVAL_POLICY_VERSION = 'v1_visual_clean_lvo_informational'

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / 'preflight_train3'
)
CASE_ROOT = PREFLIGHT_ROOT / 'cases'
SOURCE_AUDIT_PATH = PREFLIGHT_ROOT / 'synthstrip_mask_preflight_audit.csv'
SOURCE_SUMMARY_PATH = PREFLIGHT_ROOT / 'synthstrip_mask_preflight_summary.json'
APPROVED_AUDIT_PATH = PREFLIGHT_ROOT / 'synthstrip_mask_approved_audit.csv'
APPROVAL_GATE_PATH = PREFLIGHT_ROOT / 'synthstrip_mask_approval_gate.json'

EXPECTED_SUBJECTS = {
    'sub-stroke0106',
    'sub-stroke0014',
    'sub-stroke0019',
}


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def as_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() == 'true'


required_paths = [SOURCE_AUDIT_PATH, SOURCE_SUMMARY_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'SynthStrip preflight outputs are missing:\n' + '\n'.join(missing_paths)
    )

for path in [APPROVED_AUDIT_PATH, APPROVAL_GATE_PATH]:
    assert_project_path_is_safe(path)

source_audit_df = pd.read_csv(SOURCE_AUDIT_PATH)
source_summary = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))

observed_subjects = set(source_audit_df['subject_id'].astype(str))
if observed_subjects != EXPECTED_SUBJECTS or len(source_audit_df) != 3:
    raise RuntimeError(
        f'Unexpected SynthStrip preflight subjects: {sorted(observed_subjects)}'
    )
if set(source_audit_df['dev_split'].astype(str)) != {'train'}:
    raise RuntimeError('SynthStrip approval contains a non-train subject.')
if source_audit_df['target_arrays_read'].map(as_bool).any():
    raise RuntimeError('A SynthStrip preflight record reports target-array access.')

approved_rows = []
for row in source_audit_df.itertuples(index=False):
    subject_id = str(row.subject_id)
    complete_path = CASE_ROOT / subject_id / 'complete.json'
    if not complete_path.exists():
        raise FileNotFoundError(f'Missing completed case record: {complete_path}')

    complete = json.loads(complete_path.read_text(encoding='utf-8'))
    mask_path = Path(complete['mask_path'])
    qa_path = Path(complete['qa_path'])
    if not mask_path.exists() or not qa_path.exists():
        raise FileNotFoundError(f'{subject_id} mask or QA output is missing.')
    if file_sha256(mask_path) != complete.get('mask_sha256'):
        raise RuntimeError(f'{subject_id} mask hash does not match complete.json.')
    if file_sha256(qa_path) != complete.get('qa_sha256'):
        raise RuntimeError(f'{subject_id} QA hash does not match complete.json.')

    corrected_checks = {
        'train_subject_only': str(row.dev_split) == 'train',
        'target_arrays_not_read': not as_bool(row.target_arrays_read),
        'mask_volume_between_500_and_2200_ml': (
            500.0 <= float(row.brain_volume_ml) <= 2200.0
        ),
        'single_connected_component': int(row.component_count) == 1,
        'largest_component_fraction_at_least_0p995': (
            float(row.largest_component_fraction) >= 0.995
        ),
        'brain_mask_at_least_0p99_inside_crop': (
            float(row.crop_containment_fraction) >= 0.99
        ),
        'cow_coverage_at_least_0p95': (
            float(row.cow_coverage_fraction) >= 0.95
        ),
        'mask_and_qa_hashes_verified': True,
        'manual_visual_review_clean': True,
    }

    approved_rows.append({
        **row._asdict(),
        'corrected_automated_safety_passed': bool(all(corrected_checks.values())),
        'manual_visual_review_clean': True,
        'lvo_coverage_policy': 'informational_only',
        'corrected_safety_checks_json': json.dumps(corrected_checks, sort_keys=True),
        'mask_sha256': complete['mask_sha256'],
        'qa_sha256': complete['qa_sha256'],
    })

approved_df = pd.DataFrame(approved_rows)
if not bool(approved_df['corrected_automated_safety_passed'].all()):
    failed = approved_df.loc[
        ~approved_df['corrected_automated_safety_passed'],
        'subject_id',
    ].tolist()
    raise RuntimeError(f'Corrected SynthStrip safety policy failed: {failed}')

approved_df.to_csv(APPROVED_AUDIT_PATH, index=False)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'gate_passed': True,
    'approval_policy_version': APPROVAL_POLICY_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
    'synthstrip_version': SYNTHSTRIP_VERSION,
    'subjects': sorted(EXPECTED_SUBJECTS),
    'subject_count': 3,
    'all_subjects_train_only': True,
    'manual_visual_review_clean': True,
    'manual_review_statement': (
        'All three masks include the expected intracranial contents, exclude '
        'skull and extracranial islands, and show no visible truncation.'
    ),
    'lvo_coverage_policy': (
        'informational only because lvo_proximity >= 0.5 is an approximately '
        '6.93 mm exponential-distance neighborhood rather than a brain mask'
    ),
    'corrected_automated_safety_passed': True,
    'target_arrays_read': False,
    'three_case_registration_rerun_authorized': True,
    'bulk_processing_authorized': False,
    'source_audit_path': str(SOURCE_AUDIT_PATH),
    'source_audit_sha256': file_sha256(SOURCE_AUDIT_PATH),
    'source_summary_path': str(SOURCE_SUMMARY_PATH),
    'source_summary_sha256': file_sha256(SOURCE_SUMMARY_PATH),
    'approved_audit_path': str(APPROVED_AUDIT_PATH),
    'approved_audit_sha256': file_sha256(APPROVED_AUDIT_PATH),
}
APPROVAL_GATE_PATH.write_text(json.dumps(gate, indent=2), encoding='utf-8')

print('Corrected SynthStrip approval audit:')
display(approved_df[[
    'subject_id',
    'brain_volume_ml',
    'largest_component_fraction',
    'crop_containment_fraction',
    'cow_coverage_fraction',
    'lvo_coverage_fraction',
    'lvo_coverage_policy',
    'corrected_automated_safety_passed',
    'manual_visual_review_clean',
]])

print('\n' + '=' * 80)
print('TRIAL8 SYNTHSTRIP MASK APPROVAL GATE PASSED')
print('=' * 80)
print('Approved train masks:                3/3')
print('LVO proximity coverage:              informational only')
print('Three-case registration authorized:  True')
print('Bulk processing authorized:          False')
print('Study trainable parameters:          0 (preprocessing only)')
print(f'Approved audit:                       {APPROVED_AUDIT_PATH}')
print(f'Approval gate:                        {APPROVAL_GATE_PATH}')
print('\nNext: run the revised three-case masked registration preflight.')


In [ ]:
import os
import sys
import ast
import json
import time
import shutil
import hashlib
import subprocess
import tempfile
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path




ITK_THREADS = min(4, os.cpu_count() or 2)
os.environ['ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS'] = str(ITK_THREADS)


PINNED_ANTSPYX = '0.6.3'
PINNED_SCIPY = '1.16.3'
PREFLIGHT_CODE_REVISION = 'r3_synthstrip_mask_affine_vs_syn'


def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None


dependency_versions = {
    'antspyx': installed_version('antspyx'),
    'scipy': installed_version('scipy'),
}

dependency_errors = []
if dependency_versions['antspyx'] != PINNED_ANTSPYX:
    dependency_errors.append(
        f'antspyx must be {PINNED_ANTSPYX}; '
        f'found {dependency_versions["antspyx"]}'
    )
if dependency_versions['scipy'] != PINNED_SCIPY:
    dependency_errors.append(
        f'scipy must be {PINNED_SCIPY}; '
        f'found {dependency_versions["scipy"]}'
    )

if dependency_errors:
    raise RuntimeError(
        'Trial8 registration environment is not ready. Run the standalone '
        'Trial8 registration environment setup cell, allow its clean restart, '
        'then run that setup cell once more. Problems: '
        + '; '.join(dependency_errors)
    )

try:
    import numpy as np
    import pandas as pd
    import nibabel as nib
    import scipy
    from scipy import ndimage
    import matplotlib
    import matplotlib.pyplot as plt
    import ants
except Exception as exc:
    raise RuntimeError(
        'The Trial8 scientific stack could not be imported cleanly. '
        'Run the standalone registration environment setup cell and allow '
        'Colab to restart before retrying this preflight.'
    ) from exc

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')
print(f'Trial8 registration preflight revision: {PREFLIGHT_CODE_REVISION}')






PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SCHEMA_VERSION = 'vascular_hierarchy_v1'
PREFLIGHT_VERSION = 'v2_jhu_to_ncct_synra_synthstrip_mi'
SOURCE_PREFLIGHT_VERSION = 'v1_jhu_to_ncct_synra_mi'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
SEED = 20240824

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
VERIFICATION_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'final_verification'
    / 'r1_verification_gate.json'
)
VERIFICATION_CASE_AUDIT_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'final_verification'
    / 'r1_case_audit.csv'
)

ATLAS_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
)
ACQUISITION_GATE_PATH = (
    ATLAS_ROOT / 'acquisition_audit' / 'atlas_acquisition_gate.json'
)
SCHEMA_DIR = ATLAS_ROOT / 'schema' / SCHEMA_VERSION
SCHEMA_GATE_PATH = SCHEMA_DIR / 'schema_freeze_gate.json'
SCHEMA_JSON_PATH = SCHEMA_DIR / 'vascular_label_schema.json'
SCHEMA_CSV_PATH = SCHEMA_DIR / 'vascular_label_schema.csv'
LABEL_AUDIT_PATH = (
    ATLAS_ROOT
    / 'label_hierarchy_audit'
    / 'atlas_label_hierarchy_audit.json'
)
JHU_REFERENCE_PATH = (
    ATLAS_ROOT
    / 'reference'
    / 'JHU_MNI_SS'
    / 'JHU_MNI_SS_T1_brain_182x218x182.nii.gz'
)
FULL_FINE_ATLAS_PATH = (
    ATLAS_ROOT
    / 'source'
    / 'Arterial_Atlas'
    / 'data'
    / 'Atlas_182'
    / 'ArterialAtlas.nii'
)

SYNTHSTRIP_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / 'preflight_train3'
)
SYNTHSTRIP_APPROVAL_GATE_PATH = (
    SYNTHSTRIP_PREFLIGHT_ROOT / 'synthstrip_mask_approval_gate.json'
)
SYNTHSTRIP_APPROVED_AUDIT_PATH = (
    SYNTHSTRIP_PREFLIGHT_ROOT / 'synthstrip_mask_approved_audit.csv'
)
SOURCE_SELECTION_PATH = (
    ATLAS_ROOT
    / 'registration_preflight'
    / SOURCE_PREFLIGHT_VERSION
    / 'selected_train_cases.csv'
)

PREFLIGHT_ROOT = ATLAS_ROOT / 'registration_preflight' / PREFLIGHT_VERSION
CASE_ROOT = PREFLIGHT_ROOT / 'cases'
SELECTION_PATH = PREFLIGHT_ROOT / 'selected_train_cases.csv'
CONFIG_PATH = PREFLIGHT_ROOT / 'registration_preflight_config.json'
METRICS_PATH = PREFLIGHT_ROOT / 'registration_preflight_metrics.csv'
SUMMARY_PATH = PREFLIGHT_ROOT / 'registration_preflight_summary.json'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
SELECTION_QUANTILES = [0.10, 0.50, 0.90]
SELECTION_NAMES = ['small_crop', 'median_crop', 'large_crop']

REGISTRATION_CONFIG = {
    'type_of_transform': 'SyNRA',
    'aff_metric': 'mattes',
    'aff_sampling': 32,
    'aff_random_sampling_rate': 0.25,
    'syn_metric': 'mattes',
    'syn_sampling': 32,
    'reg_iterations': [80, 60, 40, 20],
    'grad_step': 0.15,
    'flow_sigma': 3,
    'total_sigma': 0,
    'mask_all_stages': True,
    'random_seed': SEED,
    'itk_threads': ITK_THREADS,
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


for path in [PREFLIGHT_ROOT, CASE_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()






required_paths = [
    FROZEN_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    VERIFICATION_CASE_AUDIT_PATH,
    ACQUISITION_GATE_PATH,
    SCHEMA_GATE_PATH,
    SCHEMA_JSON_PATH,
    SCHEMA_CSV_PATH,
    LABEL_AUDIT_PATH,
    JHU_REFERENCE_PATH,
    FULL_FINE_ATLAS_PATH,
    SYNTHSTRIP_APPROVAL_GATE_PATH,
    SYNTHSTRIP_APPROVED_AUDIT_PATH,
    SOURCE_SELECTION_PATH,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required preflight inputs:\n' + '\n'.join(missing))

verification_gate = json.loads(VERIFICATION_GATE_PATH.read_text(encoding='utf-8'))
acquisition_gate = json.loads(ACQUISITION_GATE_PATH.read_text(encoding='utf-8'))
schema_gate = json.loads(SCHEMA_GATE_PATH.read_text(encoding='utf-8'))
schema_record = json.loads(SCHEMA_JSON_PATH.read_text(encoding='utf-8'))
label_audit_record = json.loads(LABEL_AUDIT_PATH.read_text(encoding='utf-8'))
synthstrip_approval_gate = json.loads(
    SYNTHSTRIP_APPROVAL_GATE_PATH.read_text(encoding='utf-8')
)

if verification_gate.get('gate_passed') is not True:
    raise RuntimeError('Trial8 preprocessing verification gate is not passed.')
recorded_case_audit_path = Path(
    verification_gate.get('case_audit_path', '')
).resolve()
if recorded_case_audit_path != VERIFICATION_CASE_AUDIT_PATH.resolve():
    raise RuntimeError(
        'Verification gate points to an unexpected case-audit path.'
    )
if acquisition_gate.get('gate_passed') is not True:
    raise RuntimeError('Atlas acquisition gate is not passed.')
if schema_gate.get('gate_passed') is not True:
    raise RuntimeError('Atlas schema gate is not passed.')
if schema_record.get('schema_version') != SCHEMA_VERSION:
    raise RuntimeError('Unexpected atlas schema version.')
if file_sha256(SCHEMA_JSON_PATH) != schema_gate.get('schema_json_sha256'):
    raise RuntimeError('Frozen atlas schema changed after its gate was written.')
if file_sha256(FULL_FINE_ATLAS_PATH) != schema_record.get('source_fine_atlas_sha256'):
    raise RuntimeError('Fine atlas changed after schema freeze.')
if file_sha256(JHU_REFERENCE_PATH) != label_audit_record.get('jhu_reference_sha256'):
    raise RuntimeError('JHU reference changed after atlas schema freeze.')
if synthstrip_approval_gate.get('gate_passed') is not True:
    raise RuntimeError('SynthStrip mask approval gate is not passed.')
if synthstrip_approval_gate.get('manual_visual_review_clean') is not True:
    raise RuntimeError('SynthStrip masks do not have recorded visual approval.')
if synthstrip_approval_gate.get('three_case_registration_rerun_authorized') is not True:
    raise RuntimeError('Three-case masked registration rerun is not authorized.')
if synthstrip_approval_gate.get('bulk_processing_authorized') is not False:
    raise RuntimeError('Unexpected bulk authorization in SynthStrip approval gate.')
if file_sha256(SYNTHSTRIP_APPROVED_AUDIT_PATH) != synthstrip_approval_gate.get(
    'approved_audit_sha256'
):
    raise RuntimeError('Approved SynthStrip audit changed after gate creation.')

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(f'Frozen manifest has {len(manifest_df)} rows; expected 148.')
split_counts = {
    str(key): int(value)
    for key, value in manifest_df['dev_split'].value_counts().to_dict().items()
}
if split_counts != EXPECTED_SPLITS:
    raise RuntimeError(f'Unexpected split counts: {split_counts}')
if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError('Frozen manifest contains duplicate subject IDs.')

required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen manifest is missing columns: {missing_columns}')

case_audit_df = pd.read_csv(VERIFICATION_CASE_AUDIT_PATH)
required_case_audit_columns = {'subject_id', 'valid', 'image_shape'}
missing_case_audit_columns = sorted(
    required_case_audit_columns - set(case_audit_df.columns)
)
if missing_case_audit_columns:
    raise RuntimeError(
        'Verification case audit is missing columns: '
        f'{missing_case_audit_columns}'
    )
if len(case_audit_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Verification case audit has {len(case_audit_df)} rows; expected 148.'
    )
if case_audit_df['subject_id'].duplicated().any():
    raise RuntimeError('Verification case audit contains duplicate subject IDs.')

case_audit_valid = case_audit_df['valid'].map(
    lambda value: value if isinstance(value, (bool, np.bool_))
    else str(value).strip().lower() == 'true'
)
if not case_audit_valid.all():
    invalid_subjects = case_audit_df.loc[
        ~case_audit_valid,
        'subject_id',
    ].astype(str).tolist()
    raise RuntimeError(
        f'Verification case audit contains invalid subjects: {invalid_subjects}'
    )

schema_df = pd.read_csv(SCHEMA_CSV_PATH)
graph_node_mask = schema_df['use_as_graph_node'].map(
    lambda value: value if isinstance(value, (bool, np.bool_))
    else str(value).strip().lower() == 'true'
)
vascular_schema_df = schema_df[graph_node_mask].copy()
if set(vascular_schema_df['fine_label'].astype(int)) != set(range(1, 31)):
    raise RuntimeError('Frozen schema does not define graph nodes 1..30.')

fine_to_level2 = dict(
    zip(schema_df['fine_label'].astype(int), schema_df['level2_label'].astype(int))
)

print('All Trial8 preprocessing and atlas gates passed.')
print(f'Manifest subjects: {len(manifest_df)}')
print(f'Split counts: {split_counts}')
print(f'ANTsPy version: {getattr(ants, "__version__", "unknown")}')
print(f'SciPy version: {scipy.__version__}')
print(f'ITK threads: {ITK_THREADS}')
print('Registration compute: CPU (ANTsPy)')
print('Study trainable parameters: 0 (preprocessing only)')







def parse_shape(value):
    if isinstance(value, (tuple, list, np.ndarray)):
        result = tuple(int(v) for v in value)
    else:
        result = tuple(int(v) for v in ast.literal_eval(str(value)))
    if len(result) != 4:
        raise ValueError(f'Expected C x H x W x D shape, found {result}')
    return result


train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
train_df = train_df.merge(
    case_audit_df[['subject_id', 'image_shape']],
    on='subject_id',
    how='left',
    validate='one_to_one',
)
if train_df['image_shape'].isna().any():
    missing_shape_subjects = train_df.loc[
        train_df['image_shape'].isna(),
        'subject_id',
    ].astype(str).tolist()
    raise RuntimeError(
        'Audited image shape is missing for train subjects: '
        f'{missing_shape_subjects}'
    )

train_df['parsed_image_shape'] = train_df['image_shape'].map(parse_shape)
train_df['crop_voxels'] = train_df['parsed_image_shape'].map(
    lambda shape: int(np.prod(shape[1:], dtype=np.int64))
)
train_df = train_df.sort_values(['crop_voxels', 'subject_id']).reset_index(drop=True)

selected_rows = []
selected_subjects = set()
for quantile, selection_name in zip(SELECTION_QUANTILES, SELECTION_NAMES):
    target_volume = float(train_df['crop_voxels'].quantile(quantile))
    candidates = train_df[~train_df['subject_id'].isin(selected_subjects)].copy()
    candidates['distance_to_target'] = np.abs(candidates['crop_voxels'] - target_volume)
    row = candidates.sort_values(['distance_to_target', 'subject_id']).iloc[0]
    selected_subjects.add(row['subject_id'])
    selected_rows.append(
        {
            'selection_name': selection_name,
            'selection_quantile': quantile,
            'subject_id': row['subject_id'],
            'dev_split': row['dev_split'],
            'npz_path': row['vascufate_v2_npz_path'],
            'image_shape': str(row['parsed_image_shape']),
            'crop_voxels': int(row['crop_voxels']),
            'target_crop_voxels': target_volume,
        }
    )

selection_df = pd.DataFrame(selected_rows)
if set(selection_df['dev_split']) != {'train'}:
    raise RuntimeError('Registration preflight selection contains a non-train case.')
if selection_df['subject_id'].nunique() != len(SELECTION_QUANTILES):
    raise RuntimeError('Registration preflight subjects are not unique.')

source_selection_df = pd.read_csv(SOURCE_SELECTION_PATH)
if set(source_selection_df['subject_id'].astype(str)) != set(
    selection_df['subject_id'].astype(str)
):
    raise RuntimeError('Revised preflight selection differs from the original preflight.')

synthstrip_audit_df = pd.read_csv(SYNTHSTRIP_APPROVED_AUDIT_PATH)
if set(synthstrip_audit_df['subject_id'].astype(str)) != set(
    selection_df['subject_id'].astype(str)
):
    raise RuntimeError('Approved SynthStrip subjects differ from the selection.')
approved_safety = synthstrip_audit_df[
    'corrected_automated_safety_passed'
].map(
    lambda value: value if isinstance(value, (bool, np.bool_))
    else str(value).strip().lower() == 'true'
)
if not bool(approved_safety.all()):
    raise RuntimeError('An approved SynthStrip mask failed the corrected policy.')
synthstrip_records = synthstrip_audit_df.set_index('subject_id').to_dict('index')

selection_df.to_csv(SELECTION_PATH, index=False)

config_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
    'schema_version': SCHEMA_VERSION,
    'seed': SEED,
    'selection_method': 'train-only crop-volume quantiles; no target arrays read',
    'selection_shape_source': str(VERIFICATION_CASE_AUDIT_PATH),
    'selection_shape_source_sha256': file_sha256(
        VERIFICATION_CASE_AUDIT_PATH
    ),
    'selection_quantiles': SELECTION_QUANTILES,
    'registration': REGISTRATION_CONFIG,
    'fixed_image': '0.75 * ncct_brain_window + 0.25 * ncct_broad_window',
    'fixed_registration_mask': 'approved SynthStrip default-CSF mask',
    'fixed_registration_mask_gate': str(SYNTHSTRIP_APPROVAL_GATE_PATH),
    'moving_image': str(JHU_REFERENCE_PATH),
    'label_source': str(FULL_FINE_ATLAS_PATH),
    'label_interpolator': 'genericLabel',
    'label_candidates': ['affine', 'SyNRA'],
    'final_label_policy': 'clip each candidate to approved SynthStrip mask',
    'visual_qa_required': True,
    'bulk_registration_authorized': False,
    'validation_or_test_processed': False,
    'target_arrays_read': False,
}
CONFIG_PATH.write_text(json.dumps(config_record, indent=2), encoding='utf-8')

print('\nSelected train-only registration preflight cases:')
display(selection_df)







def save_nifti(data, affine, path, dtype=np.float32):
    array = np.asarray(data, dtype=dtype)
    image = nib.Nifti1Image(array, np.asarray(affine, dtype=np.float64))
    image.set_sform(np.asarray(affine, dtype=np.float64), code=2)
    image.set_qform(np.asarray(affine, dtype=np.float64), code=1)
    nib.save(image, str(path))


def dice_binary(first, second, eps=1e-6):
    first = np.asarray(first, dtype=bool)
    second = np.asarray(second, dtype=bool)
    return float(
        (2.0 * np.logical_and(first, second).sum() + eps)
        / (first.sum() + second.sum() + eps)
    )


def normalized_mutual_information(first, second, mask, bins=64):
    first = np.asarray(first, dtype=np.float32)
    second = np.asarray(second, dtype=np.float32)
    mask = np.asarray(mask, dtype=bool) & np.isfinite(first) & np.isfinite(second)
    if mask.sum() < 100:
        return float('nan')

    x = first[mask]
    y = second[mask]
    histogram, _, _ = np.histogram2d(x, y, bins=bins)
    probability = histogram / max(histogram.sum(), 1.0)
    px = probability.sum(axis=1)
    py = probability.sum(axis=0)
    nz_joint = probability > 0
    nz_x = px > 0
    nz_y = py > 0
    h_joint = -float(np.sum(probability[nz_joint] * np.log(probability[nz_joint])))
    h_x = -float(np.sum(px[nz_x] * np.log(px[nz_x])))
    h_y = -float(np.sum(py[nz_y] * np.log(py[nz_y])))
    return float((h_x + h_y) / h_joint) if h_joint > 0 else float('nan')


def center_of_mass_world(mask, affine):
    coordinates = np.argwhere(np.asarray(mask, dtype=bool))
    if len(coordinates) == 0:
        return np.full(3, np.nan, dtype=np.float64)
    center_voxel = coordinates.mean(axis=0)
    return np.asarray(nib.affines.apply_affine(affine, center_voxel), dtype=np.float64)


def homolog_pair_metrics(labels, affine):
    present_pairs = 0
    correctly_ordered_pairs = 0
    x_differences = []
    for left_label in range(1, 30, 2):
        right_label = left_label + 1
        left_mask = labels == left_label
        right_mask = labels == right_label
        if left_mask.any() and right_mask.any():
            present_pairs += 1
            left_x = center_of_mass_world(left_mask, affine)[0]
            right_x = center_of_mass_world(right_mask, affine)[0]
            difference = float(right_x - left_x)
            x_differences.append(difference)
            if difference > 0:
                correctly_ordered_pairs += 1
    return present_pairs, correctly_ordered_pairs, x_differences


def label_boundary(labels):
    labels = np.asarray(labels)
    boundary = np.zeros(labels.shape, dtype=bool)
    for axis in range(3):
        left = [slice(None)] * 3
        right = [slice(None)] * 3
        left[axis] = slice(0, -1)
        right[axis] = slice(1, None)
        difference = labels[tuple(left)] != labels[tuple(right)]
        boundary[tuple(left)] |= difference
        boundary[tuple(right)] |= difference
    return boundary & (labels > 0)


def display_slice(array, axis, index):
    if axis == 0:
        image = array[index, :, :]
    elif axis == 1:
        image = array[:, index, :]
    else:
        image = array[:, :, index]
    return np.rot90(image)


def save_visual_qa(
    fixed_array,
    affine_warped,
    nonlinear_warped,
    affine_fine_clipped,
    syn_fine_clipped,
    cow_label,
    lvo_proximity,
    fixed_mask,
    output_path,
    subject_id,
):
    center = np.rint(np.argwhere(fixed_mask).mean(axis=0)).astype(int)
    axes = [2, 1, 0]
    axis_names = ['axial', 'coronal', 'sagittal']
    indices = [int(center[axis]) for axis in axes]
    mask_boundary = fixed_mask & ~ndimage.binary_erosion(fixed_mask)
    affine_boundaries = label_boundary(affine_fine_clipped)
    syn_boundaries = label_boundary(syn_fine_clipped)
    affine_ventricles = np.isin(affine_fine_clipped, [31, 32])
    syn_ventricles = np.isin(syn_fine_clipped, [31, 32])
    vessel_mask = cow_label > 0
    lvo_mask = lvo_proximity >= 0.5

    fig, plot_axes = plt.subplots(3, 5, figsize=(23, 13))
    for row, (axis, axis_name, index) in enumerate(zip(axes, axis_names, indices)):
        fixed_slice = display_slice(fixed_array, axis, index)
        affine_slice = display_slice(affine_warped, axis, index)
        nonlinear_slice = display_slice(nonlinear_warped, axis, index)
        mask_boundary_slice = display_slice(mask_boundary, axis, index)
        affine_labels_slice = display_slice(affine_fine_clipped, axis, index)
        syn_labels_slice = display_slice(syn_fine_clipped, axis, index)
        affine_boundary_slice = display_slice(affine_boundaries, axis, index)
        syn_boundary_slice = display_slice(syn_boundaries, axis, index)
        affine_ventricle_slice = display_slice(affine_ventricles, axis, index)
        syn_ventricle_slice = display_slice(syn_ventricles, axis, index)
        vessel_slice = display_slice(vessel_mask, axis, index)
        lvo_slice = display_slice(lvo_mask, axis, index)

        plot_axes[row, 0].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 0].imshow(
            np.ma.masked_where(~mask_boundary_slice, mask_boundary_slice),
            cmap='autumn',
            alpha=0.95,
        )
        plot_axes[row, 0].set_title(f'{axis_name}: NCCT + brain mask')

        plot_axes[row, 1].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 1].imshow(affine_slice, cmap='magma', alpha=0.40)
        plot_axes[row, 1].set_title('affine JHU overlay')

        plot_axes[row, 2].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 2].imshow(nonlinear_slice, cmap='magma', alpha=0.40)
        plot_axes[row, 2].set_title('SyN JHU overlay')

        for column, labels_slice, boundary_slice, ventricle_slice, title in [
            (
                3,
                affine_labels_slice,
                affine_boundary_slice,
                affine_ventricle_slice,
                'affine territories clipped',
            ),
            (
                4,
                syn_labels_slice,
                syn_boundary_slice,
                syn_ventricle_slice,
                'SyN territories clipped',
            ),
        ]:
            plot_axes[row, column].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
            plot_axes[row, column].imshow(
                np.ma.masked_where(labels_slice == 0, labels_slice),
                cmap='turbo',
                alpha=0.28,
                vmin=1,
                vmax=32,
            )
            plot_axes[row, column].imshow(
                np.ma.masked_where(~boundary_slice, boundary_slice),
                cmap='cool',
                alpha=0.85,
            )
            if ventricle_slice.any():
                plot_axes[row, column].contour(
                    ventricle_slice,
                    levels=[0.5],
                    colors='lime',
                    linewidths=0.8,
                )
            if vessel_slice.any():
                plot_axes[row, column].contour(
                    vessel_slice,
                    levels=[0.5],
                    colors='magenta',
                    linewidths=0.7,
                )
            if lvo_slice.any():
                plot_axes[row, column].contour(
                    lvo_slice,
                    levels=[0.5],
                    colors='yellow',
                    linewidths=1.0,
                )
            plot_axes[row, column].set_title(title)

        for column in range(5):
            plot_axes[row, column].axis('off')

    fig.suptitle(
        f'{subject_id} SynthStrip-masked registration QA\n'
        'orange=brain mask, cyan=territory boundaries, lime=ventricles, '
        'magenta=CoW, yellow=LVO proximity',
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def copy_transform_list(transform_paths, destination, prefix):
    copied = []
    for index, source in enumerate(transform_paths):
        source = Path(source)
        suffix = ''.join(source.suffixes)
        if suffix == '':
            suffix = '.transform'
        target = destination / f'{prefix}_{index:02d}{suffix}'
        shutil.copy2(source, target)
        copied.append(target.name)
    return copied


def load_full_fine_labels_3d():
    image = nib.load(str(FULL_FINE_ATLAS_PATH))
    data = np.asanyarray(image.dataobj)
    if data.ndim == 4 and data.shape[-1] == 1:
        data = data[..., 0]
    if data.ndim != 3:
        raise RuntimeError(f'Unexpected fine-atlas shape: {data.shape}')
    return image, np.rint(data).astype(np.int16)


fine_source_image, fine_source_labels = load_full_fine_labels_3d()







def process_subject(selection_row):
    subject_id = str(selection_row['subject_id'])
    final_subject_dir = CASE_ROOT / subject_id
    final_complete_path = final_subject_dir / 'complete.json'

    if final_complete_path.exists():
        print(f'Reusing completed preflight case: {subject_id}')
        completed = json.loads(final_complete_path.read_text(encoding='utf-8'))
        if completed.get('preflight_version') != PREFLIGHT_VERSION:
            raise RuntimeError(
                f'{subject_id} was created by a different preflight version.'
            )
        if completed.get('dev_split') != 'train':
            raise RuntimeError(f'{subject_id} cached preflight is not train-only.')
        if completed.get('preflight_code_revision') != PREFLIGHT_CODE_REVISION:
            raise RuntimeError(f'{subject_id} cached code revision is not current.')
        cached_hashes = completed.get('output_sha256', {})
        if not cached_hashes:
            raise RuntimeError(f'{subject_id} cached outputs have no hash record.')
        for file_name, expected_hash in cached_hashes.items():
            cached_path = final_subject_dir / file_name
            if not cached_path.exists():
                raise FileNotFoundError(f'Missing cached output: {cached_path}')
            if file_sha256(cached_path) != expected_hash:
                raise RuntimeError(f'Cached output hash changed: {cached_path}')
        return completed
    if final_subject_dir.exists():
        raise RuntimeError(
            f'Incomplete preflight directory exists for {subject_id}: '
            f'{final_subject_dir}. Move it aside before rerunning.'
        )

    npz_path = Path(selection_row['npz_path'])
    if not npz_path.exists():
        raise FileNotFoundError(f'Missing Trial8 NPZ for {subject_id}: {npz_path}')

    synthstrip_record = synthstrip_records.get(subject_id)
    if synthstrip_record is None:
        raise RuntimeError(f'Missing approved SynthStrip record for {subject_id}.')
    synthstrip_mask_path = Path(synthstrip_record['mask_path'])
    if not synthstrip_mask_path.exists():
        raise FileNotFoundError(
            f'Missing approved SynthStrip mask for {subject_id}: '
            f'{synthstrip_mask_path}'
        )
    synthstrip_mask_sha256 = file_sha256(synthstrip_mask_path)
    if synthstrip_mask_sha256 != synthstrip_record['mask_sha256']:
        raise RuntimeError(f'{subject_id} SynthStrip mask hash changed after approval.')

    local_parent = Path(tempfile.mkdtemp(prefix=f'trial8_reg_{subject_id}_', dir='/tmp'))
    local_case_dir = local_parent / subject_id
    local_case_dir.mkdir(parents=True, exist_ok=False)

    start_time = time.time()
    try:
        with np.load(npz_path, allow_pickle=False) as data:
            required_keys = {
                'image',
                'crop_mask',
                'cow_label',
                'lvo_proximity',
                'input_channels',
                'affine',
                'subject_id',
            }
            missing_keys = sorted(required_keys - set(data.files))
            if missing_keys:
                raise RuntimeError(f'{subject_id} NPZ missing keys: {missing_keys}')

            stored_subject_id = str(data['subject_id'].item())
            if stored_subject_id != subject_id:
                raise RuntimeError(
                    f'NPZ subject mismatch: expected {subject_id}, found {stored_subject_id}'
                )

            channel_names = [str(value) for value in data['input_channels'].tolist()]
            brain_index = channel_names.index('ncct_brain_window')
            broad_index = channel_names.index('ncct_broad_window')

            image_array = data['image']
            image_shape = tuple(int(value) for value in image_array.shape)
            brain_window = image_array[brain_index].astype(np.float32)
            broad_window = image_array[broad_index].astype(np.float32)
            crop_mask_np = data['crop_mask'].astype(bool)
            cow_label = data['cow_label'].astype(np.uint8)
            lvo_proximity = data['lvo_proximity'].astype(np.float32)
            patient_affine = data['affine'].astype(np.float64)

        if image_shape[0] != len(channel_names):
            raise RuntimeError(f'{subject_id} channel metadata mismatch.')
        synthstrip_mask_image = nib.load(str(synthstrip_mask_path))
        synthstrip_mask_data = np.asanyarray(synthstrip_mask_image.dataobj)
        if synthstrip_mask_data.ndim == 4 and synthstrip_mask_data.shape[-1] == 1:
            synthstrip_mask_data = synthstrip_mask_data[..., 0]
        fixed_mask_np = synthstrip_mask_data > 0.5

        if brain_window.shape != fixed_mask_np.shape:
            raise RuntimeError(
                f'{subject_id} fixed image and SynthStrip mask shapes differ.'
            )
        if not np.allclose(
            synthstrip_mask_image.affine,
            patient_affine,
            atol=1e-5,
            rtol=1e-5,
        ):
            raise RuntimeError(
                f'{subject_id} SynthStrip mask affine does not match the NPZ grid.'
            )
        if ''.join(nib.aff2axcodes(patient_affine)) != 'RAS':
            raise RuntimeError(f'{subject_id} Trial8 crop is not RAS-oriented.')
        if not fixed_mask_np.any():
            raise RuntimeError(f'{subject_id} fixed registration mask is empty.')
        brain_inside_crop_fraction = float(
            np.logical_and(fixed_mask_np, crop_mask_np).sum()
            / max(fixed_mask_np.sum(), 1)
        )
        if brain_inside_crop_fraction < 0.99:
            raise RuntimeError(
                f'{subject_id} SynthStrip mask is insufficiently contained in crop.'
            )

        fixed_array = np.clip(
            0.75 * brain_window + 0.25 * broad_window,
            0.0,
            1.0,
        ).astype(np.float32)
        fixed_array[~fixed_mask_np] = 0.0

        fixed_path = local_case_dir / 'patient_ncct_fixed.nii.gz'
        fixed_mask_path = local_case_dir / 'patient_registration_mask.nii.gz'
        moving_path = local_case_dir / 'jhu_mni_ss_moving.nii.gz'
        moving_mask_path = local_case_dir / 'jhu_mni_ss_mask.nii.gz'
        full_fine_path = local_case_dir / 'arterial_atlas_full_fine_3d.nii.gz'

        save_nifti(fixed_array, patient_affine, fixed_path, np.float32)
        save_nifti(fixed_mask_np.astype(np.uint8), patient_affine, fixed_mask_path, np.uint8)
        shutil.copy2(JHU_REFERENCE_PATH, moving_path)

        jhu_image = nib.load(str(JHU_REFERENCE_PATH))
        jhu_data = np.asanyarray(jhu_image.dataobj).astype(np.float32)
        if jhu_data.ndim == 4 and jhu_data.shape[-1] == 1:
            jhu_data = jhu_data[..., 0]
        moving_mask_np = jhu_data > 0
        save_nifti(moving_mask_np.astype(np.uint8), jhu_image.affine, moving_mask_path, np.uint8)
        save_nifti(
            fine_source_labels,
            fine_source_image.affine,
            full_fine_path,
            np.int16,
        )

        fixed = ants.image_read(str(fixed_path), pixeltype='float')
        fixed_mask = ants.image_read(str(fixed_mask_path), pixeltype='unsigned char')
        moving = ants.image_read(str(moving_path), pixeltype='float')
        moving_mask = ants.image_read(str(moving_mask_path), pixeltype='unsigned char')
        moving_labels = ants.image_read(str(full_fine_path), pixeltype='float')

        print(f'\nRegistering {subject_id} with SyNRA + Mattes MI...')
        registration = ants.registration(
            fixed=fixed,
            moving=moving,
            type_of_transform=REGISTRATION_CONFIG['type_of_transform'],
            outprefix=str(local_case_dir / 'ants_'),
            mask=fixed_mask,
            moving_mask=moving_mask,
            mask_all_stages=REGISTRATION_CONFIG['mask_all_stages'],
            grad_step=REGISTRATION_CONFIG['grad_step'],
            flow_sigma=REGISTRATION_CONFIG['flow_sigma'],
            total_sigma=REGISTRATION_CONFIG['total_sigma'],
            aff_metric=REGISTRATION_CONFIG['aff_metric'],
            aff_sampling=REGISTRATION_CONFIG['aff_sampling'],
            aff_random_sampling_rate=REGISTRATION_CONFIG['aff_random_sampling_rate'],
            syn_metric=REGISTRATION_CONFIG['syn_metric'],
            syn_sampling=REGISTRATION_CONFIG['syn_sampling'],
            reg_iterations=tuple(REGISTRATION_CONFIG['reg_iterations']),
            random_seed=REGISTRATION_CONFIG['random_seed'],
            smoothing_in_mm=True,
            verbose=True,
        )

        forward_transforms = list(registration['fwdtransforms'])
        inverse_transforms = list(registration['invtransforms'])
        affine_transforms = [
            path for path in forward_transforms if str(path).lower().endswith('.mat')
        ]
        warp_transforms = [
            path
            for path in forward_transforms
            if str(path).lower().endswith(('.nii', '.nii.gz'))
            and 'inversewarp' not in str(path).lower()
        ]
        if not affine_transforms:
            raise RuntimeError(f'{subject_id} registration produced no affine transform.')
        if not warp_transforms:
            raise RuntimeError(f'{subject_id} registration produced no forward warp.')

        affine_warped = ants.apply_transforms(
            fixed=fixed,
            moving=moving,
            transformlist=affine_transforms,
            interpolator='linear',
        )
        nonlinear_warped = registration['warpedmovout']
        affine_mask = ants.apply_transforms(
            fixed=fixed,
            moving=moving_mask,
            transformlist=affine_transforms,
            interpolator='nearestNeighbor',
        )
        nonlinear_mask = ants.apply_transforms(
            fixed=fixed,
            moving=moving_mask,
            transformlist=forward_transforms,
            interpolator='nearestNeighbor',
        )
        affine_labels_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving_labels,
            transformlist=affine_transforms,
            interpolator='genericLabel',
        )
        syn_labels_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving_labels,
            transformlist=forward_transforms,
            interpolator='genericLabel',
        )

        affine_warped_np = affine_warped.numpy().astype(np.float32)
        nonlinear_warped_np = nonlinear_warped.numpy().astype(np.float32)
        affine_mask_np = affine_mask.numpy() > 0.5
        nonlinear_mask_np = nonlinear_mask.numpy() > 0.5
        affine_full_fine_raw = np.rint(
            affine_labels_image.numpy()
        ).astype(np.int16)
        syn_full_fine_raw = np.rint(syn_labels_image.numpy()).astype(np.int16)

        unexpected_labels = sorted(
            (
                set(int(value) for value in np.unique(affine_full_fine_raw))
                | set(int(value) for value in np.unique(syn_full_fine_raw))
            )
            - set(range(33))
        )
        affine_full_fine_clipped = affine_full_fine_raw.copy()
        syn_full_fine_clipped = syn_full_fine_raw.copy()
        affine_full_fine_clipped[~fixed_mask_np] = 0
        syn_full_fine_clipped[~fixed_mask_np] = 0

        affine_vascular_fine = affine_full_fine_clipped.copy()
        syn_vascular_fine = syn_full_fine_clipped.copy()
        affine_vascular_fine[np.isin(affine_vascular_fine, [31, 32])] = 0
        syn_vascular_fine[np.isin(syn_vascular_fine, [31, 32])] = 0

        affine_level2 = np.zeros_like(affine_full_fine_clipped, dtype=np.int16)
        syn_level2 = np.zeros_like(syn_full_fine_clipped, dtype=np.int16)
        for fine_label, parent_label in fine_to_level2.items():
            affine_level2[affine_full_fine_clipped == fine_label] = parent_label
            syn_level2[syn_full_fine_clipped == fine_label] = parent_label

        affine_output_path = local_case_dir / 'jhu_affine_warped.nii.gz'
        nonlinear_output_path = local_case_dir / 'jhu_syn_warped.nii.gz'
        affine_raw_fine_path = (
            local_case_dir / 'arterial_atlas_full_fine_affine_raw.nii.gz'
        )
        syn_raw_fine_path = (
            local_case_dir / 'arterial_atlas_full_fine_syn_raw.nii.gz'
        )
        affine_clipped_fine_path = (
            local_case_dir / 'arterial_atlas_full_fine_affine_clipped.nii.gz'
        )
        syn_clipped_fine_path = (
            local_case_dir / 'arterial_atlas_full_fine_syn_clipped.nii.gz'
        )
        affine_vascular_path = (
            local_case_dir / 'arterial_atlas_vascular30_affine_clipped.nii.gz'
        )
        syn_vascular_path = (
            local_case_dir / 'arterial_atlas_vascular30_syn_clipped.nii.gz'
        )
        affine_level2_path = (
            local_case_dir / 'arterial_atlas_level2_affine_clipped.nii.gz'
        )
        syn_level2_path = (
            local_case_dir / 'arterial_atlas_level2_syn_clipped.nii.gz'
        )
        jacobian_output_path = local_case_dir / 'syn_jacobian_determinant.nii.gz'
        qa_output_path = local_case_dir / 'registration_visual_qa.png'

        save_nifti(affine_warped_np, patient_affine, affine_output_path, np.float32)
        save_nifti(nonlinear_warped_np, patient_affine, nonlinear_output_path, np.float32)
        save_nifti(affine_full_fine_raw, patient_affine, affine_raw_fine_path, np.int16)
        save_nifti(syn_full_fine_raw, patient_affine, syn_raw_fine_path, np.int16)
        save_nifti(
            affine_full_fine_clipped,
            patient_affine,
            affine_clipped_fine_path,
            np.int16,
        )
        save_nifti(
            syn_full_fine_clipped,
            patient_affine,
            syn_clipped_fine_path,
            np.int16,
        )
        save_nifti(affine_vascular_fine, patient_affine, affine_vascular_path, np.int16)
        save_nifti(syn_vascular_fine, patient_affine, syn_vascular_path, np.int16)
        save_nifti(affine_level2, patient_affine, affine_level2_path, np.int16)
        save_nifti(syn_level2, patient_affine, syn_level2_path, np.int16)

        jacobian_image = ants.create_jacobian_determinant_image(
            fixed,
            warp_transforms[0],
            do_log=False,
            geom=False,
        )
        jacobian_np = jacobian_image.numpy().astype(np.float32)
        save_nifti(jacobian_np, patient_affine, jacobian_output_path, np.float32)

        fixed_mask_count = int(fixed_mask_np.sum())
        affine_atlas_support_raw = affine_full_fine_raw > 0
        syn_atlas_support_raw = syn_full_fine_raw > 0
        affine_atlas_support_count = int(affine_atlas_support_raw.sum())
        syn_atlas_support_count = int(syn_atlas_support_raw.sum())
        affine_inside_count = int(
            np.logical_and(affine_atlas_support_raw, fixed_mask_np).sum()
        )
        syn_inside_count = int(
            np.logical_and(syn_atlas_support_raw, fixed_mask_np).sum()
        )
        affine_inside_fraction = (
            affine_inside_count / affine_atlas_support_count
            if affine_atlas_support_count else 0.0
        )
        syn_inside_fraction = (
            syn_inside_count / syn_atlas_support_count
            if syn_atlas_support_count else 0.0
        )

        overlap_mask_affine = fixed_mask_np & affine_mask_np
        overlap_mask_nonlinear = fixed_mask_np & nonlinear_mask_np
        affine_nmi = normalized_mutual_information(
            fixed_array,
            affine_warped_np,
            overlap_mask_affine,
        )
        nonlinear_nmi = normalized_mutual_information(
            fixed_array,
            nonlinear_warped_np,
            overlap_mask_nonlinear,
        )

        fixed_center = center_of_mass_world(fixed_mask_np, patient_affine)
        affine_center = center_of_mass_world(affine_mask_np, patient_affine)
        nonlinear_center = center_of_mass_world(nonlinear_mask_np, patient_affine)

        jacobian_in_mask = jacobian_np[fixed_mask_np]
        jacobian_nonpositive_fraction = float(
            np.mean(jacobian_in_mask <= 0)
        )

        affine_present_vascular_labels = sorted(
            int(value)
            for value in np.unique(affine_vascular_fine)
            if 1 <= int(value) <= 30
        )
        syn_present_vascular_labels = sorted(
            int(value)
            for value in np.unique(syn_vascular_fine)
            if 1 <= int(value) <= 30
        )
        (
            affine_present_homolog_pairs,
            affine_correctly_ordered_pairs,
            affine_homolog_x_differences,
        ) = homolog_pair_metrics(affine_full_fine_clipped, patient_affine)
        (
            syn_present_homolog_pairs,
            syn_correctly_ordered_pairs,
            syn_homolog_x_differences,
        ) = homolog_pair_metrics(syn_full_fine_clipped, patient_affine)

        save_visual_qa(
            fixed_array=fixed_array,
            affine_warped=affine_warped_np,
            nonlinear_warped=nonlinear_warped_np,
            affine_fine_clipped=affine_full_fine_clipped,
            syn_fine_clipped=syn_full_fine_clipped,
            cow_label=cow_label,
            lvo_proximity=lvo_proximity,
            fixed_mask=fixed_mask_np,
            output_path=qa_output_path,
            subject_id=subject_id,
        )

        copied_forward = copy_transform_list(
            forward_transforms,
            local_case_dir,
            'forward_transform',
        )
        copied_inverse = copy_transform_list(
            inverse_transforms,
            local_case_dir,
            'inverse_transform',
        )

        output_sha256 = {
            path.name: file_sha256(path)
            for path in sorted(local_case_dir.iterdir())
            if path.is_file()
        }

        elapsed_seconds = float(time.time() - start_time)
        metrics = {
            'preflight_version': PREFLIGHT_VERSION,
            'preflight_code_revision': PREFLIGHT_CODE_REVISION,
            'subject_id': subject_id,
            'selection_name': str(selection_row['selection_name']),
            'dev_split': 'train',
            'npz_path': str(npz_path),
            'synthstrip_mask_path': str(synthstrip_mask_path),
            'synthstrip_mask_sha256': synthstrip_mask_sha256,
            'fixed_shape': list(fixed_array.shape),
            'fixed_orientation': ''.join(nib.aff2axcodes(patient_affine)),
            'fixed_mask_voxels': fixed_mask_count,
            'brain_inside_crop_fraction': brain_inside_crop_fraction,
            'affine_mask_dice': dice_binary(fixed_mask_np, affine_mask_np),
            'nonlinear_mask_dice': dice_binary(fixed_mask_np, nonlinear_mask_np),
            'affine_nmi': affine_nmi,
            'nonlinear_nmi': nonlinear_nmi,
            'affine_center_distance_mm': float(np.linalg.norm(fixed_center - affine_center)),
            'nonlinear_center_distance_mm': float(np.linalg.norm(fixed_center - nonlinear_center)),
            'affine_raw_atlas_support_voxels': affine_atlas_support_count,
            'syn_raw_atlas_support_voxels': syn_atlas_support_count,
            'affine_raw_atlas_inside_brain_fraction': float(
                affine_inside_fraction
            ),
            'syn_raw_atlas_inside_brain_fraction': float(syn_inside_fraction),
            'affine_raw_atlas_clipped_fraction': float(1.0 - affine_inside_fraction),
            'syn_raw_atlas_clipped_fraction': float(1.0 - syn_inside_fraction),
            'affine_present_vascular_label_count': len(
                affine_present_vascular_labels
            ),
            'syn_present_vascular_label_count': len(syn_present_vascular_labels),
            'affine_present_vascular_labels': affine_present_vascular_labels,
            'syn_present_vascular_labels': syn_present_vascular_labels,
            'syn_ventricle_labels_present': [
                label
                for label in [31, 32]
                if np.any(syn_full_fine_clipped == label)
            ],
            'unexpected_labels': unexpected_labels,
            'affine_present_homolog_pairs': affine_present_homolog_pairs,
            'affine_correctly_ordered_homolog_pairs': (
                affine_correctly_ordered_pairs
            ),
            'syn_present_homolog_pairs': syn_present_homolog_pairs,
            'syn_correctly_ordered_homolog_pairs': syn_correctly_ordered_pairs,
            'affine_minimum_right_minus_left_centroid_mm': float(
                min(affine_homolog_x_differences)
            ) if affine_homolog_x_differences else float('nan'),
            'syn_minimum_right_minus_left_centroid_mm': float(
                min(syn_homolog_x_differences)
            ) if syn_homolog_x_differences else float('nan'),
            'jacobian_min': float(np.min(jacobian_in_mask)),
            'jacobian_p01': float(np.quantile(jacobian_in_mask, 0.01)),
            'jacobian_median': float(np.median(jacobian_in_mask)),
            'jacobian_p99': float(np.quantile(jacobian_in_mask, 0.99)),
            'jacobian_max': float(np.max(jacobian_in_mask)),
            'jacobian_nonpositive_fraction': jacobian_nonpositive_fraction,
            'forward_transform_files': copied_forward,
            'inverse_transform_files': copied_inverse,
            'visual_qa_file': qa_output_path.name,
            'affine_clipped_fine_file': affine_clipped_fine_path.name,
            'syn_clipped_fine_file': syn_clipped_fine_path.name,
            'affine_clipped_vascular_file': affine_vascular_path.name,
            'syn_clipped_vascular_file': syn_vascular_path.name,
            'affine_clipped_level2_file': affine_level2_path.name,
            'syn_clipped_level2_file': syn_level2_path.name,
            'elapsed_seconds': elapsed_seconds,
            'target_arrays_read': False,
            'output_sha256': output_sha256,
        }

        safety_checks = {
            'train_subject_only': True,
            'fixed_orientation_is_ras': metrics['fixed_orientation'] == 'RAS',
            'no_unexpected_labels': len(unexpected_labels) == 0,
            'brain_mask_at_least_0p99_inside_crop': (
                brain_inside_crop_fraction >= 0.99
            ),
            'affine_at_least_24_of_30_vascular_labels_present': (
                len(affine_present_vascular_labels) >= 24
            ),
            'syn_at_least_24_of_30_vascular_labels_present': (
                len(syn_present_vascular_labels) >= 24
            ),
            'affine_at_least_12_homolog_pairs_present': (
                affine_present_homolog_pairs >= 12
            ),
            'syn_at_least_12_homolog_pairs_present': (
                syn_present_homolog_pairs >= 12
            ),
            'affine_pairs_have_correct_left_right_order': (
                affine_correctly_ordered_pairs == affine_present_homolog_pairs
            ),
            'syn_pairs_have_correct_left_right_order': (
                syn_correctly_ordered_pairs == syn_present_homolog_pairs
            ),
            'no_nonpositive_jacobians_in_fixed_mask': (
                jacobian_nonpositive_fraction <= 1e-6
            ),
            'affine_at_least_80_percent_raw_atlas_inside_brain': (
                affine_inside_fraction >= 0.80
            ),
            'syn_at_least_80_percent_raw_atlas_inside_brain': (
                syn_inside_fraction >= 0.80
            ),
        }
        metrics['automated_safety_checks'] = safety_checks
        metrics['automated_safety_passed'] = bool(all(safety_checks.values()))
        metrics['visual_qa_approved'] = False
        metrics['bulk_registration_authorized'] = False

        complete_path = local_case_dir / 'complete.json'
        complete_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

        shutil.copytree(local_case_dir, final_subject_dir)
        print(
            f'Completed {subject_id}: '
            f'safety={metrics["automated_safety_passed"]}, '
            f'affine_labels={len(affine_present_vascular_labels)}/30, '
            f'SyN_labels={len(syn_present_vascular_labels)}/30, '
            f'Jac<=0={jacobian_nonpositive_fraction:.6g}, '
            f'{elapsed_seconds / 60:.1f} min'
        )
        return metrics
    finally:
        if local_parent.exists():
            shutil.rmtree(local_parent)






metrics_rows = []
for index, row in selection_df.iterrows():
    print('\n' + '=' * 80)
    print(
        f'Registration preflight {index + 1}/{len(selection_df)}: '
        f'{row["subject_id"]} ({row["selection_name"]})'
    )
    print('=' * 80)
    metrics_rows.append(process_subject(row))

flat_metric_rows = []
for metrics in metrics_rows:
    flat_metric_rows.append(
        {
            key: value
            for key, value in metrics.items()
            if not isinstance(value, (dict, list))
        }
    )

metrics_df = pd.DataFrame(flat_metric_rows)
metrics_df.to_csv(METRICS_PATH, index=False)

all_train = all(row['dev_split'] == 'train' for row in metrics_rows)
all_automated_safety = all(
    bool(row['automated_safety_passed']) for row in metrics_rows
)
all_visual_files_exist = all(
    (CASE_ROOT / row['subject_id'] / row['visual_qa_file']).exists()
    for row in metrics_rows
)

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'subjects': [row['subject_id'] for row in metrics_rows],
    'all_subjects_are_train': all_train,
    'all_automated_safety_checks_passed': all_automated_safety,
    'all_visual_qa_files_exist': all_visual_files_exist,
    'visual_qa_approved': False,
    'bulk_registration_authorized': False,
    'validation_or_test_processed': False,
    'target_arrays_read': False,
    'metrics_path': str(METRICS_PATH),
    'case_root': str(CASE_ROOT),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('\nRegistration preflight metrics:')
display(metrics_df)

print('\n' + '=' * 80)
print('TRIAL8 SYNTHSTRIP-MASKED REGISTRATION PREFLIGHT COMPLETE')
print('=' * 80)
print(f'All subjects train-only:         {all_train}')
print(f'Automated safety passed:         {all_automated_safety}')
print(f'Visual QA files created:         {all_visual_files_exist}')
print('Visual QA approved:              False')
print('Bulk registration authorized:    False')
print(f'Metrics:                         {METRICS_PATH}')
print(f'Case outputs:                    {CASE_ROOT}')
print(f'Summary:                         {SUMMARY_PATH}')
print('\nNo validation/test subjects or target arrays were used.')
print('Stop here and check every visual QA image before bulk registration.')

if not all_train or not all_visual_files_exist:
    raise RuntimeError(
        'Registration preflight output integrity failed. Do not continue.'
    )

if not all_automated_safety:
    raise RuntimeError(
        'One or more automated registration safety checks failed. '
        'Do not continue to bulk registration; check the saved metrics and QA.'
    )


In [ ]:
from pathlib import Path

import pandas as pd
from PIL import Image
from IPython.display import display, Markdown
from google.colab import drive


drive.mount('/mnt/storage')
print('Study trainable parameters: 0 (preprocessing review only)')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
PREFLIGHT_VERSION = 'v2_jhu_to_ncct_synra_synthstrip_mi'

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'registration_preflight'
    / PREFLIGHT_VERSION
)
METRICS_PATH = PREFLIGHT_ROOT / 'registration_preflight_metrics.csv'
CASE_ROOT = PREFLIGHT_ROOT / 'cases'

if not METRICS_PATH.exists():
    raise FileNotFoundError(f'Missing preflight metrics: {METRICS_PATH}')
if not CASE_ROOT.exists():
    raise FileNotFoundError(f'Missing preflight case directory: {CASE_ROOT}')

metrics_df = pd.read_csv(METRICS_PATH)

expected_subjects = {
    'sub-stroke0106',
    'sub-stroke0014',
    'sub-stroke0019',
}
observed_subjects = set(metrics_df['subject_id'].astype(str))
if observed_subjects != expected_subjects:
    raise RuntimeError(
        'Unexpected preflight subjects. '
        f'Expected {sorted(expected_subjects)}, found {sorted(observed_subjects)}.'
    )

required_columns = {
    'subject_id',
    'selection_name',
    'affine_mask_dice',
    'nonlinear_mask_dice',
    'affine_nmi',
    'nonlinear_nmi',
    'affine_center_distance_mm',
    'nonlinear_center_distance_mm',
    'brain_inside_crop_fraction',
    'affine_raw_atlas_inside_brain_fraction',
    'syn_raw_atlas_inside_brain_fraction',
    'affine_raw_atlas_clipped_fraction',
    'syn_raw_atlas_clipped_fraction',
    'affine_present_vascular_label_count',
    'syn_present_vascular_label_count',
    'affine_present_homolog_pairs',
    'affine_correctly_ordered_homolog_pairs',
    'syn_present_homolog_pairs',
    'syn_correctly_ordered_homolog_pairs',
    'jacobian_min',
    'jacobian_p01',
    'jacobian_median',
    'jacobian_p99',
    'jacobian_max',
    'jacobian_nonpositive_fraction',
    'automated_safety_passed',
    'target_arrays_read',
}
missing_columns = sorted(required_columns - set(metrics_df.columns))
if missing_columns:
    raise RuntimeError(f'Metrics are missing columns: {missing_columns}')

metrics_df['mask_dice_change_syn_minus_affine'] = (
    metrics_df['nonlinear_mask_dice'] - metrics_df['affine_mask_dice']
)
metrics_df['nmi_change_syn_minus_affine'] = (
    metrics_df['nonlinear_nmi'] - metrics_df['affine_nmi']
)
metrics_df['center_distance_change_syn_minus_affine_mm'] = (
    metrics_df['nonlinear_center_distance_mm']
    - metrics_df['affine_center_distance_mm']
)

review_columns = [
    'subject_id',
    'selection_name',
    'affine_mask_dice',
    'nonlinear_mask_dice',
    'mask_dice_change_syn_minus_affine',
    'affine_nmi',
    'nonlinear_nmi',
    'nmi_change_syn_minus_affine',
    'affine_center_distance_mm',
    'nonlinear_center_distance_mm',
    'center_distance_change_syn_minus_affine_mm',
    'brain_inside_crop_fraction',
    'affine_raw_atlas_inside_brain_fraction',
    'syn_raw_atlas_inside_brain_fraction',
    'affine_raw_atlas_clipped_fraction',
    'syn_raw_atlas_clipped_fraction',
    'affine_present_vascular_label_count',
    'syn_present_vascular_label_count',
    'affine_present_homolog_pairs',
    'affine_correctly_ordered_homolog_pairs',
    'syn_present_homolog_pairs',
    'syn_correctly_ordered_homolog_pairs',
    'jacobian_min',
    'jacobian_p01',
    'jacobian_median',
    'jacobian_p99',
    'jacobian_max',
    'jacobian_nonpositive_fraction',
    'automated_safety_passed',
    'target_arrays_read',
]

print('Trial8 SynthStrip-masked registration metric review:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    260,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(
        metrics_df[review_columns]
        .sort_values('selection_name')
        .reset_index(drop=True)
    )

checklist = [
    'The orange SynthStrip boundary follows the intracranial boundary.',
    'Patient brain anatomy agrees with both JHU intensity overlays.',
    'Midline, ventricles, and major sulcal geometry remain plausible.',
    'Both clipped territory candidates remain inside the brain mask.',
    'Left and right territories are not exchanged.',
    'Posterior and infratentorial territories remain anatomically plausible.',
    'CoW and LVO contours are plausible relative to vascular territories.',
    'SyN improves local alignment without stretching, pinching, or folding.',
    'Affine is preferred if SyN adds distortion without anatomical benefit.',
]

display(Markdown('## Visual review checklist'))
for item in checklist:
    print(f'[ ] {item}')

for row in metrics_df.sort_values('selection_name').itertuples(index=False):
    qa_path = CASE_ROOT / str(row.subject_id) / 'registration_visual_qa.png'
    if not qa_path.exists():
        raise FileNotFoundError(f'Missing visual QA image: {qa_path}')

    display(Markdown(
        f'## {row.subject_id}: {row.selection_name}\n\n'
        f'- Affine mask Dice: `{row.affine_mask_dice:.4f}`\n'
        f'- SyN mask Dice: `{row.nonlinear_mask_dice:.4f}`\n'
        f'- Affine NMI: `{row.affine_nmi:.4f}`\n'
        f'- SyN NMI: `{row.nonlinear_nmi:.4f}`\n'
        f'- Affine raw atlas inside brain: '
        f'`{row.affine_raw_atlas_inside_brain_fraction:.4f}`\n'
        f'- SyN raw atlas inside brain: '
        f'`{row.syn_raw_atlas_inside_brain_fraction:.4f}`\n'
        f'- Affine / SyN clipped fractions: '
        f'`{row.affine_raw_atlas_clipped_fraction:.4f} / '
        f'{row.syn_raw_atlas_clipped_fraction:.4f}`\n'
        f'- Affine / SyN vascular labels: '
        f'`{int(row.affine_present_vascular_label_count)} / '
        f'{int(row.syn_present_vascular_label_count)}`\n'
        f'- Jacobian p01 / median / p99: '
        f'`{row.jacobian_p01:.4f} / {row.jacobian_median:.4f} / '
        f'{row.jacobian_p99:.4f}`'
    ))

    with Image.open(qa_path) as qa_image:
        display(qa_image.copy())

print('\n' + '=' * 80)
print('MASKED REGISTRATION VISUAL QA REVIEW COMPLETE')
print('=' * 80)
print('No approval flags or bulk-registration authorization were changed.')
print('Check all three panels before selecting affine or SyN.')


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
PREFLIGHT_VERSION = 'v2_jhu_to_ncct_synra_synthstrip_mi'
DIAGNOSTIC_VERSION = 'v1_full_and_vascular_coverage'

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'registration_preflight'
    / PREFLIGHT_VERSION
)
CASE_ROOT = PREFLIGHT_ROOT / 'cases'
METRICS_PATH = PREFLIGHT_ROOT / 'registration_preflight_metrics.csv'
DIAGNOSTIC_ROOT = PREFLIGHT_ROOT / 'brain_coverage_diagnostic'
AUDIT_PATH = DIAGNOSTIC_ROOT / 'atlas_brain_coverage_audit.csv'
SUMMARY_PATH = DIAGNOSTIC_ROOT / 'atlas_brain_coverage_summary.json'

EXPECTED_SUBJECTS = {
    'sub-stroke0106',
    'sub-stroke0014',
    'sub-stroke0019',
}
CORE_DEPTH_MM = 3.0


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def as_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() == 'true'


def load_3d(path):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 3:
        raise RuntimeError(f'Expected a 3D image at {path}, found {array.shape}.')
    return image, array


def display_slice(array, axis, index):
    return np.rot90(np.take(array, int(index), axis=axis))


def boundary(mask):
    mask = np.asarray(mask, dtype=bool)
    return mask & ~ndimage.binary_erosion(mask)


def coverage_metrics(labels, brain_mask, brain_core, spacing):
    labels = np.asarray(labels)
    full_support = labels > 0
    vascular_support = (labels >= 1) & (labels <= 30)
    ventricle_support = np.isin(labels, [31, 32])
    outside_support = full_support & ~brain_mask
    uncovered_brain = brain_mask & ~full_support
    uncovered_core = brain_core & ~full_support

    brain_voxels = int(brain_mask.sum())
    core_voxels = int(brain_core.sum())
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)

    if full_support.any():
        distance_to_label = ndimage.distance_transform_edt(
            ~full_support,
            sampling=spacing,
        )
        uncovered_distances = distance_to_label[uncovered_brain]
    else:
        uncovered_distances = np.asarray([], dtype=np.float64)

    return {
        'full_brain_coverage_fraction': float(
            np.logical_and(full_support, brain_mask).sum()
            / max(brain_voxels, 1)
        ),
        'vascular_brain_coverage_fraction': float(
            np.logical_and(vascular_support, brain_mask).sum()
            / max(brain_voxels, 1)
        ),
        'ventricle_brain_fraction': float(
            np.logical_and(ventricle_support, brain_mask).sum()
            / max(brain_voxels, 1)
        ),
        'core_full_coverage_fraction': float(
            np.logical_and(full_support, brain_core).sum()
            / max(core_voxels, 1)
        ),
        'uncovered_brain_voxels': int(uncovered_brain.sum()),
        'uncovered_brain_volume_ml': float(
            uncovered_brain.sum() * voxel_volume_ml
        ),
        'uncovered_core_voxels': int(uncovered_core.sum()),
        'outside_brain_label_voxels': int(outside_support.sum()),
        'maximum_uncovered_distance_mm': float(
            uncovered_distances.max()
        ) if len(uncovered_distances) else 0.0,
        'p95_uncovered_distance_mm': float(
            np.quantile(uncovered_distances, 0.95)
        ) if len(uncovered_distances) else 0.0,
        'full_support': full_support,
        'uncovered_brain': uncovered_brain,
        'outside_support': outside_support,
    }


def save_gap_qa(
    fixed_array,
    brain_mask,
    affine_labels,
    syn_labels,
    affine_result,
    syn_result,
    output_path,
    subject_id,
):
    brain_boundary = boundary(brain_mask)
    gap_union = affine_result['uncovered_brain'] | syn_result['uncovered_brain']
    axes = [2, 1, 0]
    axis_names = ['axial', 'coronal', 'sagittal']

    fig, panels = plt.subplots(3, 3, figsize=(16, 13))
    for row, (axis, axis_name) in enumerate(zip(axes, axis_names)):
        other_axes = tuple(index for index in range(3) if index != axis)
        per_slice_gap = gap_union.sum(axis=other_axes)
        index = int(np.argmax(per_slice_gap))

        fixed_slice = display_slice(fixed_array, axis, index)
        boundary_slice = display_slice(brain_boundary, axis, index)

        panels[row, 0].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 0].imshow(
            np.ma.masked_where(~boundary_slice, boundary_slice),
            cmap='autumn',
            alpha=0.95,
        )
        panels[row, 0].set_title(f'{axis_name}: brain-mask boundary')

        for column, labels, result, title in [
            (1, affine_labels, affine_result, 'affine coverage gaps'),
            (2, syn_labels, syn_result, 'SyN coverage gaps'),
        ]:
            labels_slice = display_slice(labels, axis, index)
            gap_slice = display_slice(result['uncovered_brain'], axis, index)
            outside_slice = display_slice(result['outside_support'], axis, index)

            panels[row, column].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
            panels[row, column].imshow(
                np.ma.masked_where(labels_slice == 0, labels_slice),
                cmap='turbo',
                alpha=0.25,
                vmin=1,
                vmax=32,
            )
            panels[row, column].imshow(
                np.ma.masked_where(~gap_slice, gap_slice),
                cmap='Reds',
                alpha=0.80,
            )
            panels[row, column].imshow(
                np.ma.masked_where(~outside_slice, outside_slice),
                cmap='Purples',
                alpha=0.95,
            )
            panels[row, column].set_title(title)

        for column in range(3):
            panels[row, column].axis('off')

    fig.suptitle(
        f'{subject_id} atlas-to-brain coverage QA\n'
        'red=unlabeled intracranial voxels, purple=labels outside brain',
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


if not METRICS_PATH.exists():
    raise FileNotFoundError(f'Missing masked registration metrics: {METRICS_PATH}')

assert_project_path_is_safe(DIAGNOSTIC_ROOT)
DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)

registration_df = pd.read_csv(METRICS_PATH)
observed_subjects = set(registration_df['subject_id'].astype(str))
if observed_subjects != EXPECTED_SUBJECTS or len(registration_df) != 3:
    raise RuntimeError(
        f'Unexpected masked-registration subjects: {sorted(observed_subjects)}'
    )
if registration_df['target_arrays_read'].map(as_bool).any():
    raise RuntimeError('A masked-registration record reports target-array access.')

audit_rows = []

for row in registration_df.itertuples(index=False):
    subject_id = str(row.subject_id)
    case_dir = CASE_ROOT / subject_id
    fixed_path = case_dir / 'patient_ncct_fixed.nii.gz'
    brain_mask_path = case_dir / 'patient_registration_mask.nii.gz'
    affine_path = case_dir / str(row.affine_clipped_fine_file)
    syn_path = case_dir / str(row.syn_clipped_fine_file)
    qa_path = DIAGNOSTIC_ROOT / f'{subject_id}_atlas_brain_coverage_qa.png'

    required_paths = [fixed_path, brain_mask_path, affine_path, syn_path]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            f'{subject_id} is missing diagnostic inputs:\n' + '\n'.join(missing_paths)
        )

    fixed_image, fixed_array = load_3d(fixed_path)
    mask_image, mask_array = load_3d(brain_mask_path)
    affine_image, affine_labels = load_3d(affine_path)
    syn_image, syn_labels = load_3d(syn_path)

    reference_shape = fixed_array.shape
    reference_affine = fixed_image.affine
    for name, image, array in [
        ('brain mask', mask_image, mask_array),
        ('affine labels', affine_image, affine_labels),
        ('SyN labels', syn_image, syn_labels),
    ]:
        if array.shape != reference_shape:
            raise RuntimeError(
                f'{subject_id} {name} shape {array.shape} differs from '
                f'{reference_shape}.'
            )
        if not np.allclose(image.affine, reference_affine, atol=1e-5, rtol=1e-5):
            raise RuntimeError(f'{subject_id} {name} affine mismatch.')

    brain_mask = mask_array > 0.5
    spacing = np.sqrt(np.sum(reference_affine[:3, :3] ** 2, axis=0))
    brain_depth = ndimage.distance_transform_edt(brain_mask, sampling=spacing)
    brain_core = brain_mask & (brain_depth >= CORE_DEPTH_MM)

    affine_result = coverage_metrics(
        affine_labels,
        brain_mask,
        brain_core,
        spacing,
    )
    syn_result = coverage_metrics(
        syn_labels,
        brain_mask,
        brain_core,
        spacing,
    )

    save_gap_qa(
        fixed_array=fixed_array,
        brain_mask=brain_mask,
        affine_labels=affine_labels,
        syn_labels=syn_labels,
        affine_result=affine_result,
        syn_result=syn_result,
        output_path=qa_path,
        subject_id=subject_id,
    )

    audit_rows.append({
        'subject_id': subject_id,
        'selection_name': str(row.selection_name),
        'brain_mask_voxels': int(brain_mask.sum()),
        'brain_core_voxels': int(brain_core.sum()),
        'affine_full_brain_coverage_fraction': (
            affine_result['full_brain_coverage_fraction']
        ),
        'syn_full_brain_coverage_fraction': (
            syn_result['full_brain_coverage_fraction']
        ),
        'affine_vascular_brain_coverage_fraction': (
            affine_result['vascular_brain_coverage_fraction']
        ),
        'syn_vascular_brain_coverage_fraction': (
            syn_result['vascular_brain_coverage_fraction']
        ),
        'affine_core_full_coverage_fraction': (
            affine_result['core_full_coverage_fraction']
        ),
        'syn_core_full_coverage_fraction': (
            syn_result['core_full_coverage_fraction']
        ),
        'affine_uncovered_brain_volume_ml': (
            affine_result['uncovered_brain_volume_ml']
        ),
        'syn_uncovered_brain_volume_ml': (
            syn_result['uncovered_brain_volume_ml']
        ),
        'affine_uncovered_core_voxels': (
            affine_result['uncovered_core_voxels']
        ),
        'syn_uncovered_core_voxels': syn_result['uncovered_core_voxels'],
        'affine_p95_uncovered_distance_mm': (
            affine_result['p95_uncovered_distance_mm']
        ),
        'syn_p95_uncovered_distance_mm': (
            syn_result['p95_uncovered_distance_mm']
        ),
        'affine_maximum_uncovered_distance_mm': (
            affine_result['maximum_uncovered_distance_mm']
        ),
        'syn_maximum_uncovered_distance_mm': (
            syn_result['maximum_uncovered_distance_mm']
        ),
        'affine_outside_brain_label_voxels': (
            affine_result['outside_brain_label_voxels']
        ),
        'syn_outside_brain_label_voxels': (
            syn_result['outside_brain_label_voxels']
        ),
        'qa_path': str(qa_path),
        'target_arrays_read': False,
        'bulk_processing_authorized': False,
    })

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(AUDIT_PATH, index=False)

outside_zero = bool(
    (audit_df['affine_outside_brain_label_voxels'] == 0).all()
    and (audit_df['syn_outside_brain_label_voxels'] == 0).all()
)

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'diagnostic_version': DIAGNOSTIC_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'subjects': sorted(EXPECTED_SUBJECTS),
    'core_depth_mm': CORE_DEPTH_MM,
    'all_clipped_label_maps_have_zero_outside_brain_voxels': outside_zero,
    'minimum_affine_full_brain_coverage_fraction': float(
        audit_df['affine_full_brain_coverage_fraction'].min()
    ),
    'minimum_syn_full_brain_coverage_fraction': float(
        audit_df['syn_full_brain_coverage_fraction'].min()
    ),
    'minimum_affine_core_full_coverage_fraction': float(
        audit_df['affine_core_full_coverage_fraction'].min()
    ),
    'minimum_syn_core_full_coverage_fraction': float(
        audit_df['syn_core_full_coverage_fraction'].min()
    ),
    'target_arrays_read': False,
    'transform_selected': False,
    'bulk_processing_authorized': False,
    'audit_path': str(AUDIT_PATH),
    'qa_directory': str(DIAGNOSTIC_ROOT),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Trial8 atlas-to-brain coverage diagnostic:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    280,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(audit_df)

print('\n' + '=' * 80)
print('TRIAL8 ATLAS-TO-BRAIN COVERAGE DIAGNOSTIC COMPLETE')
print('=' * 80)
print(f'Zero clipped labels outside brain:  {outside_zero}')
print(
    'Minimum affine full coverage:     '
    f'{summary["minimum_affine_full_brain_coverage_fraction"]:.4f}'
)
print(
    'Minimum SyN full coverage:        '
    f'{summary["minimum_syn_full_brain_coverage_fraction"]:.4f}'
)
print(
    'Minimum affine core coverage:     '
    f'{summary["minimum_affine_core_full_coverage_fraction"]:.4f}'
)
print(
    'Minimum SyN core coverage:        '
    f'{summary["minimum_syn_core_full_coverage_fraction"]:.4f}'
)
print('Study trainable parameters:        0 (diagnostic only)')
print('Target arrays read:                 False')
print('Transform selected:                 False')
print('Bulk processing authorized:         False')
print(f'Audit:                              {AUDIT_PATH}')
print(f'QA directory:                       {DIAGNOSTIC_ROOT}')
print(f'Summary:                            {SUMMARY_PATH}')
print('\nStop here and check the metrics and red gap overlays.')


In [ ]:
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
PREFLIGHT_VERSION = 'v2_jhu_to_ncct_synra_synthstrip_mi'
SELECTION_VERSION = 'v1_synra_synthstrip_clip_valid'

PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'registration_preflight'
    / PREFLIGHT_VERSION
)
REGISTRATION_METRICS_PATH = (
    PREFLIGHT_ROOT / 'registration_preflight_metrics.csv'
)
REGISTRATION_SUMMARY_PATH = (
    PREFLIGHT_ROOT / 'registration_preflight_summary.json'
)
COVERAGE_ROOT = PREFLIGHT_ROOT / 'brain_coverage_diagnostic'
COVERAGE_AUDIT_PATH = COVERAGE_ROOT / 'atlas_brain_coverage_audit.csv'
COVERAGE_SUMMARY_PATH = COVERAGE_ROOT / 'atlas_brain_coverage_summary.json'
SELECTION_GATE_PATH = PREFLIGHT_ROOT / 'transform_selection_gate.json'

EXPECTED_SUBJECTS = {
    'sub-stroke0106',
    'sub-stroke0014',
    'sub-stroke0019',
}
MINIMUM_FULL_COVERAGE = 0.85
MINIMUM_CORE_COVERAGE = 0.90


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == 'true'


required_paths = [
    REGISTRATION_METRICS_PATH,
    REGISTRATION_SUMMARY_PATH,
    COVERAGE_AUDIT_PATH,
    COVERAGE_SUMMARY_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 transform-selection inputs are missing:\n'
        + '\n'.join(missing_paths)
    )

registration_df = pd.read_csv(REGISTRATION_METRICS_PATH)
coverage_df = pd.read_csv(COVERAGE_AUDIT_PATH)
registration_summary = json.loads(
    REGISTRATION_SUMMARY_PATH.read_text(encoding='utf-8')
)
coverage_summary = json.loads(
    COVERAGE_SUMMARY_PATH.read_text(encoding='utf-8')
)

for name, frame in [
    ('registration', registration_df),
    ('coverage', coverage_df),
]:
    observed = set(frame['subject_id'].astype(str))
    if observed != EXPECTED_SUBJECTS or len(frame) != 3:
        raise RuntimeError(f'Unexpected {name} subjects: {sorted(observed)}')
    if frame['target_arrays_read'].map(as_bool).any():
        raise RuntimeError(f'{name} audit reports target-array access.')

if registration_summary.get('all_subjects_are_train') is not True:
    raise RuntimeError('Registration preflight was not train-only.')
if registration_summary.get('all_automated_safety_checks_passed') is not True:
    raise RuntimeError('Registration automated safety did not pass.')
if coverage_summary.get(
    'all_clipped_label_maps_have_zero_outside_brain_voxels'
) is not True:
    raise RuntimeError('A clipped atlas contains labels outside the brain.')

merged = registration_df.merge(
    coverage_df,
    on=['subject_id', 'selection_name'],
    how='inner',
    validate='one_to_one',
    suffixes=('_registration', '_coverage'),
)

decision_checks = {
    'three_train_only_cases': len(merged) == 3,
    'registration_safety_passed': bool(
        registration_df['automated_safety_passed'].map(as_bool).all()
    ),
    'all_30_syn_vascular_labels_preserved': bool(
        (registration_df['syn_present_vascular_label_count'] == 30).all()
    ),
    'all_15_syn_homolog_pairs_preserved': bool(
        (registration_df['syn_present_homolog_pairs'] == 15).all()
    ),
    'all_syn_homolog_pairs_correctly_ordered': bool(
        (
            registration_df['syn_present_homolog_pairs']
            == registration_df['syn_correctly_ordered_homolog_pairs']
        ).all()
    ),
    'no_nonpositive_syn_jacobians': bool(
        (registration_df['jacobian_nonpositive_fraction'] <= 1e-6).all()
    ),
    'zero_syn_labels_outside_brain': bool(
        (coverage_df['syn_outside_brain_label_voxels'] == 0).all()
    ),
    'minimum_syn_full_coverage_met': bool(
        (
            coverage_df['syn_full_brain_coverage_fraction']
            >= MINIMUM_FULL_COVERAGE
        ).all()
    ),
    'minimum_syn_core_coverage_met': bool(
        (
            coverage_df['syn_core_full_coverage_fraction']
            >= MINIMUM_CORE_COVERAGE
        ).all()
    ),
    'manual_quantitative_review_accepted': True,
}

gate_passed = bool(all(decision_checks.values()))
if not gate_passed:
    failed = [key for key, value in decision_checks.items() if not value]
    raise RuntimeError(f'Transform selection gate failed: {failed}')

decision_table = merged[[
    'subject_id',
    'selection_name',
    'affine_mask_dice',
    'nonlinear_mask_dice',
    'affine_nmi',
    'nonlinear_nmi',
    'affine_full_brain_coverage_fraction',
    'syn_full_brain_coverage_fraction',
    'affine_core_full_coverage_fraction',
    'syn_core_full_coverage_fraction',
    'syn_outside_brain_label_voxels',
    'jacobian_nonpositive_fraction',
]].copy()

gate_core = {
    'gate_passed': True,
    'selection_version': SELECTION_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'decision_basis': 'three train-only cases; no infarct targets read',
    'selected_registration': 'SyNRA',
    'registration_metric': 'Mattes mutual information',
    'fixed_registration_mask': 'approved SynthStrip default-CSF mask',
    'label_interpolation': 'genericLabel',
    'label_postprocessing': 'clip to approved SynthStrip brain mask',
    'graph_projection_policy': (
        'apply graph conditioning only where vascular labels 1..30 are valid; '
        'retain unmodified CNN features elsewhere'
    ),
    'atlas_valid_definition': 'clipped fine vascular label in [1, 30]',
    'minimum_full_coverage_threshold': MINIMUM_FULL_COVERAGE,
    'minimum_core_coverage_threshold': MINIMUM_CORE_COVERAGE,
    'decision_checks': decision_checks,
    'subjects': sorted(EXPECTED_SUBJECTS),
    'target_arrays_read': False,
    'bulk_synthstrip_mask_generation_authorized': True,
    'bulk_atlas_registration_authorized': False,
    'registration_metrics_path': str(REGISTRATION_METRICS_PATH),
    'registration_metrics_sha256': file_sha256(REGISTRATION_METRICS_PATH),
    'registration_summary_path': str(REGISTRATION_SUMMARY_PATH),
    'registration_summary_sha256': file_sha256(REGISTRATION_SUMMARY_PATH),
    'coverage_audit_path': str(COVERAGE_AUDIT_PATH),
    'coverage_audit_sha256': file_sha256(COVERAGE_AUDIT_PATH),
    'coverage_summary_path': str(COVERAGE_SUMMARY_PATH),
    'coverage_summary_sha256': file_sha256(COVERAGE_SUMMARY_PATH),
}
decision_fingerprint = hashlib.sha256(
    json.dumps(gate_core, sort_keys=True).encode('utf-8')
).hexdigest()

created_at_utc = datetime.now(timezone.utc).isoformat()
if SELECTION_GATE_PATH.exists():
    existing_gate = json.loads(
        SELECTION_GATE_PATH.read_text(encoding='utf-8')
    )
    if existing_gate.get('decision_fingerprint') == decision_fingerprint:
        created_at_utc = existing_gate['created_at_utc']

gate = {
    'created_at_utc': created_at_utc,
    'decision_fingerprint': decision_fingerprint,
    **gate_core,
}
SELECTION_GATE_PATH.write_text(json.dumps(gate, indent=2), encoding='utf-8')

print('Trial8 transform-selection evidence:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    260,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(decision_table)

print('\nDecision checks:')
display(pd.DataFrame([
    {'check': key, 'passed': value}
    for key, value in decision_checks.items()
]))

print('\n' + '=' * 80)
print('TRIAL8 REGISTRATION POLICY FROZEN')
print('=' * 80)
print('Selected registration:               SyNRA')
print('Fixed mask:                          SynthStrip default-CSF')
print('Final labels:                        intracranially clipped')
print('Graph projection:                    atlas-valid gated')
print('Bulk SynthStrip generation:          authorized')
print('Bulk atlas registration:             not authorized')
print('Study trainable parameters:          0 (preprocessing only)')
print('Target arrays read:                   False')
print(f'Selection gate:                       {SELECTION_GATE_PATH}')
print('\nNext: generate and audit SynthStrip masks for all 148 subjects.')


In [ ]:
import sys
import json
import time
import shutil
import hashlib
import tempfile
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
BULK_VERSION = 'bulk_all148_v1'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
SELECTION_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'registration_preflight'
    / 'v2_jhu_to_ncct_synra_synthstrip_mi'
    / 'transform_selection_gate.json'
)

SYNTHSTRIP_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
)
SOURCE_DIR = SYNTHSTRIP_ROOT / 'source'
PROVENANCE_PATH = SOURCE_DIR / 'synthstrip_source_provenance.json'
SCRIPT_PATH = SOURCE_DIR / 'mri_synthstrip'
MODEL_PATH = SOURCE_DIR / 'synthstrip.1.pt'

BULK_ROOT = SYNTHSTRIP_ROOT / BULK_VERSION
CASE_ROOT = BULK_ROOT / 'cases'
AUDIT_PATH = BULK_ROOT / 'bulk_synthstrip_mask_audit.csv'
FAILURE_PATH = BULK_ROOT / 'bulk_synthstrip_mask_failures.csv'
SUMMARY_PATH = BULK_ROOT / 'bulk_synthstrip_mask_summary.json'

VENV_ROOT = Path('/tmp/acf_net_synthstrip_py310')
VENV_PYTHON = VENV_ROOT / 'bin' / 'python'
VENV_READY_PATH = VENV_ROOT / 'trial8_synthstrip_environment.json'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)
BORDER_MM = 1



MIN_BRAIN_VOLUME_ML = 500.0
MAX_BRAIN_VOLUME_ML = 2200.0
MIN_LARGEST_COMPONENT_FRACTION = 0.995
MIN_CROP_CONTAINMENT = 0.99
MIN_COW_COVERAGE = 0.95


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def atomic_write_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)


def atomic_save_mask(mask, affine, path):
    temporary = path.with_name(path.name + '.tmp.nii.gz')
    image = nib.Nifti1Image(
        np.asarray(mask, dtype=np.uint8),
        np.asarray(affine, dtype=np.float64),
    )
    image.set_sform(np.asarray(affine, dtype=np.float64), code=2)
    image.set_qform(np.asarray(affine, dtype=np.float64), code=1)
    nib.save(image, str(temporary))
    temporary.replace(path)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == 'true'


def scalar_record(record):
    return {
        key: value
        for key, value in record.items()
        if not isinstance(value, (dict, list))
    }


def resume_record_is_valid(record, row, npz_path, mask_path, hashes):
    required = {
        'subject_id',
        'dev_split',
        'npz_path',
        'npz_sha256',
        'npz_size_bytes',
        'script_sha256',
        'model_sha256',
        'frozen_manifest_sha256',
        'selection_gate_sha256',
        'mask_path',
        'mask_sha256',
        'target_arrays_read',
        'completed',
    }
    if not required.issubset(record):
        return False
    if not mask_path.exists():
        return False
    checks = [
        record['subject_id'] == str(row.subject_id),
        record['dev_split'] == str(row.dev_split),
        record['npz_path'] == str(npz_path),
        int(record['npz_size_bytes']) == int(npz_path.stat().st_size),
        record['script_sha256'] == hashes['script'],
        record['model_sha256'] == hashes['model'],
        record['frozen_manifest_sha256'] == hashes['manifest'],
        record['selection_gate_sha256'] == hashes['selection'],
        record['mask_path'] == str(mask_path),
        record['target_arrays_read'] is False,
        record['completed'] is True,
    ]
    if not all(checks):
        return False
    if record['npz_sha256'] != file_sha256(npz_path):
        return False
    if record['mask_sha256'] != file_sha256(mask_path):
        return False
    return True


required_paths = [
    FROZEN_MANIFEST_PATH,
    SELECTION_GATE_PATH,
    PROVENANCE_PATH,
    SCRIPT_PATH,
    MODEL_PATH,
    VENV_PYTHON,
    VENV_READY_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 bulk SynthStrip prerequisites are missing. On a fresh runtime, '
        'run the standalone SynthStrip environment setup first. Missing:\n'
        + '\n'.join(missing_paths)
    )

selection_gate = json.loads(SELECTION_GATE_PATH.read_text(encoding='utf-8'))
if selection_gate.get('gate_passed') is not True:
    raise RuntimeError('Trial8 transform-selection gate did not pass.')
if selection_gate.get('selected_registration') != 'SyNRA':
    raise RuntimeError('Unexpected frozen registration selection.')
if selection_gate.get('bulk_synthstrip_mask_generation_authorized') is not True:
    raise RuntimeError('Bulk SynthStrip mask generation is not authorized.')
if selection_gate.get('bulk_atlas_registration_authorized') is not False:
    raise RuntimeError('Bulk atlas registration must remain locked at this stage.')

provenance = json.loads(PROVENANCE_PATH.read_text(encoding='utf-8'))
script_sha256 = file_sha256(SCRIPT_PATH)
model_sha256 = file_sha256(MODEL_PATH)
manifest_sha256 = file_sha256(FROZEN_MANIFEST_PATH)
selection_gate_sha256 = file_sha256(SELECTION_GATE_PATH)

if provenance.get('script_sha256') != script_sha256:
    raise RuntimeError('SynthStrip script does not match frozen provenance.')
if provenance.get('model_sha256') != model_sha256:
    raise RuntimeError('SynthStrip model does not match frozen provenance.')

gpu_result = subprocess.run(
    [
        str(VENV_PYTHON),
        '-c',
        'import torch; '
        'print(torch.cuda.get_device_name(0) if torch.cuda.is_available() '
        'else "CUDA unavailable"); '
        'raise SystemExit(0 if torch.cuda.is_available() else 1)',
    ],
    text=True,
    capture_output=True,
)
if gpu_result.returncode != 0:
    raise RuntimeError(
        'CUDA is unavailable in the isolated SynthStrip environment.\n'
        + gpu_result.stdout
        + '\n'
        + gpu_result.stderr
    )

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
required_columns = {'subject_id', 'dev_split', 'vascufate_v2_npz_path'}
missing_columns = sorted(required_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen manifest missing columns: {missing_columns}')
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Frozen manifest has {len(manifest_df)} subjects; '
        f'expected {EXPECTED_SUBJECTS}.'
    )
if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError('Frozen manifest contains duplicate subject identifiers.')

split_counts = manifest_df['dev_split'].value_counts().to_dict()
if split_counts != EXPECTED_SPLITS:
    raise RuntimeError(f'Unexpected split counts: {split_counts}')

manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
for path in [BULK_ROOT, CASE_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

hashes = {
    'script': script_sha256,
    'model': model_sha256,
    'manifest': manifest_sha256,
    'selection': selection_gate_sha256,
}

print('Trial8 bulk SynthStrip mask generation')
print(f'Compute: GPU')
print(f'GPU: {gpu_result.stdout.strip()}')
print(f'Frozen subjects: {len(manifest_df)}')
print(f'Split counts: {split_counts}')
print('Study trainable parameters: 0 (preprocessing only)')
print('Target arrays read: False')
print('Bulk atlas registration authorized: False')
print(f'Output: {BULK_ROOT}')

audit_rows = []
failure_rows = []
reused_count = 0
computed_count = 0
run_start = time.time()

for case_index, row in enumerate(manifest_df.itertuples(index=False), start=1):
    subject_id = str(row.subject_id)
    dev_split = str(row.dev_split)
    npz_path = Path(row.vascufate_v2_npz_path)
    case_dir = CASE_ROOT / subject_id
    mask_path = case_dir / 'ncct_synthstrip_mask.nii.gz'
    complete_path = case_dir / 'complete.json'
    case_dir.mkdir(parents=True, exist_ok=True)

    try:
        if not npz_path.exists():
            raise FileNotFoundError(f'NPZ not found: {npz_path}')

        if complete_path.exists() and mask_path.exists():
            existing_record = json.loads(
                complete_path.read_text(encoding='utf-8')
            )
            if resume_record_is_valid(
                existing_record,
                row,
                npz_path,
                mask_path,
                hashes,
            ):
                audit_rows.append(scalar_record(existing_record))
                reused_count += 1
                if case_index == 1 or case_index % 10 == 0:
                    print(
                        f'Reused {case_index}/{EXPECTED_SUBJECTS}: '
                        f'{subject_id}'
                    )
                continue

            print(f'Resume check failed for {subject_id}; recomputing.')

        npz_sha256 = file_sha256(npz_path)
        with np.load(npz_path, allow_pickle=False) as data:
            required_keys = {
                'image',
                'input_channels',
                'crop_mask',
                'cow_label',
                'lvo_proximity',
                'affine',
                'subject_id',
            }
            missing_keys = sorted(required_keys - set(data.files))
            if missing_keys:
                raise RuntimeError(f'NPZ missing keys: {missing_keys}')
            if str(data['subject_id'].item()) != subject_id:
                raise RuntimeError('NPZ subject identifier mismatch.')

            channel_names = [
                str(value) for value in data['input_channels'].tolist()
            ]
            if 'ncct_broad_window' not in channel_names:
                raise RuntimeError('ncct_broad_window channel is missing.')
            broad_index = channel_names.index('ncct_broad_window')

            broad_window = data['image'][broad_index].astype(np.float32)
            crop_mask = data['crop_mask'].astype(bool)
            cow_mask = data['cow_label'] > 0
            lvo_mask = data['lvo_proximity'] >= 0.5
            affine = data['affine'].astype(np.float64)

        if broad_window.ndim != 3:
            raise RuntimeError(
                f'NCCT broad-window shape is {broad_window.shape}, not 3D.'
            )
        for name, array in [
            ('crop_mask', crop_mask),
            ('cow_label', cow_mask),
            ('lvo_proximity', lvo_mask),
        ]:
            if array.shape != broad_window.shape:
                raise RuntimeError(
                    f'{name} shape {array.shape} does not match '
                    f'{broad_window.shape}.'
                )

        spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
        orientation = ''.join(nib.aff2axcodes(affine))
        if orientation != 'RAS':
            raise RuntimeError(f'Orientation is {orientation}, not RAS.')
        if not np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0):
            raise RuntimeError(f'Spacing is {spacing.tolist()}.')

        case_start = time.time()
        with tempfile.TemporaryDirectory(
            prefix=f'trial8_synthstrip_{subject_id}_',
            dir='/tmp',
        ) as temporary_directory:
            temporary_directory = Path(temporary_directory)
            input_path = temporary_directory / 'ncct_broad_window.nii.gz'
            brain_path = temporary_directory / 'ncct_brain.nii.gz'
            temporary_mask_path = temporary_directory / 'ncct_mask.nii.gz'

            input_image = nib.Nifti1Image(broad_window, affine)
            input_image.set_sform(affine, code=2)
            input_image.set_qform(affine, code=1)
            nib.save(input_image, str(input_path))

            command = [
                str(VENV_PYTHON),
                str(SCRIPT_PATH),
                '-i',
                str(input_path),
                '-o',
                str(brain_path),
                '-m',
                str(temporary_mask_path),
                '-b',
                str(BORDER_MM),
                '-g',
                '--model',
                str(MODEL_PATH),
            ]
            result = subprocess.run(
                command,
                text=True,
                capture_output=True,
            )
            if result.returncode != 0:
                raise RuntimeError(
                    'SynthStrip failed.\nSTDOUT:\n'
                    + result.stdout[-4000:]
                    + '\nSTDERR:\n'
                    + result.stderr[-4000:]
                )

            mask_image = nib.load(str(temporary_mask_path))
            mask_array = np.asanyarray(mask_image.dataobj)
            if mask_array.ndim == 4 and mask_array.shape[-1] == 1:
                mask_array = mask_array[..., 0]
            brain_mask = mask_array > 0.5

            if brain_mask.shape != broad_window.shape:
                raise RuntimeError(
                    f'Mask shape {brain_mask.shape} does not match '
                    f'{broad_window.shape}.'
                )
            if not np.allclose(
                mask_image.affine,
                affine,
                atol=1e-5,
                rtol=1e-5,
            ):
                raise RuntimeError('Mask affine does not match the NPZ grid.')

            atomic_save_mask(brain_mask, affine, mask_path)

        elapsed_seconds = float(time.time() - case_start)

        labeled, component_count = ndimage.label(brain_mask)
        component_sizes = np.bincount(labeled.ravel())[1:]
        largest_component_fraction = (
            float(component_sizes.max() / brain_mask.sum())
            if len(component_sizes) and brain_mask.any()
            else 0.0
        )

        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        brain_voxels = int(brain_mask.sum())
        brain_volume_ml = float(brain_voxels * voxel_volume_ml)
        crop_containment = float(
            np.logical_and(brain_mask, crop_mask).sum()
            / max(brain_voxels, 1)
        )
        cow_voxels = int(cow_mask.sum())
        cow_coverage = (
            float(np.logical_and(cow_mask, brain_mask).sum() / cow_voxels)
            if cow_voxels
            else 1.0
        )
        lvo_voxels = int(lvo_mask.sum())
        lvo_coverage = (
            float(np.logical_and(lvo_mask, brain_mask).sum() / lvo_voxels)
            if lvo_voxels
            else 1.0
        )
        edge_touch_x = bool(brain_mask[0].any() or brain_mask[-1].any())
        edge_touch_y = bool(
            brain_mask[:, 0].any() or brain_mask[:, -1].any()
        )
        edge_touch_z = bool(
            brain_mask[:, :, 0].any() or brain_mask[:, :, -1].any()
        )

        safety_checks = {
            'mask_nonempty': bool(brain_mask.any()),
            'brain_volume_between_500_and_2200_ml': bool(
                MIN_BRAIN_VOLUME_ML
                <= brain_volume_ml
                <= MAX_BRAIN_VOLUME_ML
            ),
            'largest_component_fraction_at_least_0p995': bool(
                largest_component_fraction
                >= MIN_LARGEST_COMPONENT_FRACTION
            ),
            'brain_mask_at_least_0p99_inside_crop_mask': bool(
                crop_containment >= MIN_CROP_CONTAINMENT
            ),
            'cow_coverage_at_least_0p95': bool(
                cow_coverage >= MIN_COW_COVERAGE
            ),
            'does_not_touch_x_or_y_crop_edges': bool(
                not (edge_touch_x or edge_touch_y)
            ),
        }

        record = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'completed': True,
            'subject_id': subject_id,
            'dev_split': dev_split,
            'npz_path': str(npz_path),
            'npz_size_bytes': int(npz_path.stat().st_size),
            'npz_sha256': npz_sha256,
            'mask_path': str(mask_path),
            'mask_sha256': file_sha256(mask_path),
            'script_sha256': script_sha256,
            'model_sha256': model_sha256,
            'frozen_manifest_sha256': manifest_sha256,
            'selection_gate_sha256': selection_gate_sha256,
            'orientation': orientation,
            'spacing_x_mm': float(spacing[0]),
            'spacing_y_mm': float(spacing[1]),
            'spacing_z_mm': float(spacing[2]),
            'shape_x': int(brain_mask.shape[0]),
            'shape_y': int(brain_mask.shape[1]),
            'shape_z': int(brain_mask.shape[2]),
            'brain_voxels': brain_voxels,
            'brain_volume_ml': brain_volume_ml,
            'component_count': int(component_count),
            'largest_component_fraction': largest_component_fraction,
            'crop_containment_fraction': crop_containment,
            'cow_voxels': cow_voxels,
            'cow_coverage_fraction': cow_coverage,
            'lvo_proximity_voxels': lvo_voxels,
            'lvo_coverage_fraction_informational': lvo_coverage,
            'edge_touch_x': edge_touch_x,
            'edge_touch_y': edge_touch_y,
            'edge_touch_z': edge_touch_z,
            'elapsed_seconds': elapsed_seconds,
            'compute': 'GPU',
            'no_csf': False,
            'border_mm': BORDER_MM,
            'target_arrays_read': False,
            'safety_checks': safety_checks,
            'automated_safety_passed': bool(all(safety_checks.values())),
            'visual_qa_approved': False,
            'bulk_atlas_registration_authorized': False,
        }
        atomic_write_json(record, complete_path)
        audit_rows.append(scalar_record(record))
        computed_count += 1

        pd.DataFrame(audit_rows).sort_values('subject_id').to_csv(
            AUDIT_PATH,
            index=False,
        )

        print(
            f'Completed {case_index}/{EXPECTED_SUBJECTS}: {subject_id}, '
            f'volume={brain_volume_ml:.1f} mL, '
            f'CoW={cow_coverage:.4f}, '
            f'safety={record["automated_safety_passed"]}, '
            f'{elapsed_seconds:.1f} s'
        )

    except Exception as error:
        failure = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'subject_id': subject_id,
            'dev_split': dev_split,
            'npz_path': str(npz_path),
            'error_type': type(error).__name__,
            'error': str(error),
            'target_arrays_read': False,
        }
        failure_rows.append(failure)
        pd.DataFrame(failure_rows).to_csv(FAILURE_PATH, index=False)
        print(
            f'FAILED {case_index}/{EXPECTED_SUBJECTS}: '
            f'{subject_id}: {type(error).__name__}: {error}'
        )

    if case_index % 10 == 0 or case_index == EXPECTED_SUBJECTS:
        elapsed_minutes = (time.time() - run_start) / 60.0
        print(
            f'Progress: {case_index}/{EXPECTED_SUBJECTS}; '
            f'computed={computed_count}, reused={reused_count}, '
            f'failed={len(failure_rows)}, elapsed={elapsed_minutes:.1f} min'
        )

audit_df = pd.DataFrame(audit_rows)
if not audit_df.empty:
    audit_df = audit_df.sort_values('subject_id').reset_index(drop=True)
    audit_df.to_csv(AUDIT_PATH, index=False)

failure_df = pd.DataFrame(failure_rows)
if not failure_df.empty:
    failure_df.to_csv(FAILURE_PATH, index=False)
elif FAILURE_PATH.exists():
    FAILURE_PATH.unlink()

all_subjects_completed = (
    len(audit_df) == EXPECTED_SUBJECTS
    and audit_df['subject_id'].nunique() == EXPECTED_SUBJECTS
)
all_automated_safety_passed = bool(
    all_subjects_completed
    and audit_df['automated_safety_passed'].map(as_bool).all()
)

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preprocessing_version': PREPROCESSING_VERSION,
    'synthstrip_version': SYNTHSTRIP_VERSION,
    'bulk_version': BULK_VERSION,
    'expected_subjects': EXPECTED_SUBJECTS,
    'completed_subjects': int(len(audit_df)),
    'computed_this_run': int(computed_count),
    'reused_this_run': int(reused_count),
    'failed_this_run': int(len(failure_df)),
    'all_subjects_completed': bool(all_subjects_completed),
    'all_automated_safety_passed': all_automated_safety_passed,
    'visual_qa_approved': False,
    'target_arrays_read': False,
    'bulk_atlas_registration_authorized': False,
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'frozen_manifest_sha256': manifest_sha256,
    'selection_gate_path': str(SELECTION_GATE_PATH),
    'selection_gate_sha256': selection_gate_sha256,
    'source_provenance_path': str(PROVENANCE_PATH),
    'script_sha256': script_sha256,
    'model_sha256': model_sha256,
    'audit_path': str(AUDIT_PATH),
    'failure_path': str(FAILURE_PATH) if failure_df.size else None,
    'case_root': str(CASE_ROOT),
    'elapsed_seconds_this_run': float(time.time() - run_start),
}
atomic_write_json(summary, SUMMARY_PATH)

print('\nBulk SynthStrip audit summary:')
if not audit_df.empty:
    summary_table = audit_df.groupby('dev_split', as_index=False).agg(
        subjects=('subject_id', 'count'),
        safety_passed=('automated_safety_passed', 'sum'),
        mean_brain_volume_ml=('brain_volume_ml', 'mean'),
        minimum_cow_coverage=('cow_coverage_fraction', 'min'),
        minimum_crop_containment=('crop_containment_fraction', 'min'),
    )
    with pd.option_context(
        'display.max_columns',
        None,
        'display.width',
        220,
        'display.float_format',
        lambda value: f'{value:.6f}',
    ):
        display(summary_table)

if not failure_df.empty:
    print('\nFailures from this run:')
    display(failure_df)

print('\n' + '=' * 80)
print('TRIAL8 BULK SYNTHSTRIP MASK GENERATION COMPLETE')
print('=' * 80)
print(f'Completed subjects:                 {len(audit_df)}/{EXPECTED_SUBJECTS}')
print(f'Computed this run:                  {computed_count}')
print(f'Reused this run:                    {reused_count}')
print(f'Failures this run:                  {len(failure_df)}')
print(f'All automated safety passed:        {all_automated_safety_passed}')
print('Study trainable parameters:         0 (preprocessing only)')
print('Target arrays read:                  False')
print('Bulk atlas registration authorized: False')
print(f'Audit:                               {AUDIT_PATH}')
print(f'Summary:                             {SUMMARY_PATH}')
print(f'Case masks:                          {CASE_ROOT}')

if not all_subjects_completed or not failure_df.empty:
    raise RuntimeError(
        'Bulk SynthStrip generation is incomplete. Rerun this same cell to '
        'resume failed or missing cases, then check the saved audit.'
    )

if not all_automated_safety_passed:
    flagged = audit_df.loc[
        ~audit_df['automated_safety_passed'].map(as_bool),
        [
            'subject_id',
            'dev_split',
            'brain_volume_ml',
            'largest_component_fraction',
            'crop_containment_fraction',
            'cow_coverage_fraction',
            'edge_touch_x',
            'edge_touch_y',
        ],
    ]
    print('\nCases requiring mask review:')
    display(flagged)
    raise RuntimeError(
        'All masks were generated, but one or more automated safety checks '
        'failed. Do not begin atlas registration. Check the flagged cases.'
    )

print('\nStop here and check the audit before running mask verification.')


In [ ]:
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
BULK_VERSION = 'bulk_all148_v1'
VERIFICATION_VERSION = 'v1_structural_mask_and_cow_compatibility'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
SELECTION_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'registration_preflight'
    / 'v2_jhu_to_ncct_synra_synthstrip_mi'
    / 'transform_selection_gate.json'
)
BULK_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / BULK_VERSION
)
CASE_ROOT = BULK_ROOT / 'cases'
GENERATION_AUDIT_PATH = BULK_ROOT / 'bulk_synthstrip_mask_audit.csv'
GENERATION_SUMMARY_PATH = BULK_ROOT / 'bulk_synthstrip_mask_summary.json'

VERIFICATION_ROOT = BULK_ROOT / 'verification' / VERIFICATION_VERSION
QA_ROOT = VERIFICATION_ROOT / 'visual_qa'
CASE_AUDIT_PATH = VERIFICATION_ROOT / 'mask_structural_audit.csv'
COW_AUDIT_PATH = VERIFICATION_ROOT / 'cow_compatibility_audit.csv'
COW_LABEL_AUDIT_PATH = VERIFICATION_ROOT / 'cow_label_compatibility_audit.csv'
QA_SELECTION_PATH = VERIFICATION_ROOT / 'visual_qa_selection.csv'
SUMMARY_PATH = VERIFICATION_ROOT / 'verification_summary.json'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)

MIN_BRAIN_VOLUME_ML = 500.0
MAX_BRAIN_VOLUME_ML = 2200.0
MIN_LARGEST_COMPONENT_FRACTION = 0.995
MIN_CROP_CONTAINMENT = 0.99
COW_REVIEW_THRESHOLD = 0.95
COW_GROSS_MISMATCH_THRESHOLD = 0.80
BOUNDARY_SPILL_P95_MM = 5.0


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def atomic_write_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == 'true'


def display_slice(array, axis, index):
    return np.rot90(np.take(array, int(index), axis=axis))


def mask_boundary(mask):
    mask = np.asarray(mask, dtype=bool)
    return mask & ~ndimage.binary_erosion(mask, iterations=1)


def select_axis_index(mask, axis, fallback_mask):
    reduce_axes = tuple(index for index in range(3) if index != axis)
    counts = np.asarray(mask, dtype=np.int64).sum(axis=reduce_axes)
    if counts.max() > 0:
        return int(np.argmax(counts))
    coordinates = np.argwhere(fallback_mask)
    if len(coordinates):
        return int(np.rint(np.median(coordinates[:, axis])))
    return int(fallback_mask.shape[axis] // 2)


def cow_category(cow_voxels, coverage, outside_p95_mm):
    if cow_voxels == 0:
        return 'cow_absent'
    if coverage >= COW_REVIEW_THRESHOLD:
        return 'strong_overlap'
    if coverage < COW_GROSS_MISMATCH_THRESHOLD:
        return 'gross_mismatch'
    if np.isfinite(outside_p95_mm) and outside_p95_mm <= BOUNDARY_SPILL_P95_MM:
        return 'near_boundary_spill'
    return 'moderate_mismatch'


def load_case_arrays(subject_id, npz_path, mask_path):
    with np.load(npz_path, allow_pickle=False) as data:
        required_keys = {
            'image',
            'input_channels',
            'crop_mask',
            'cow_label',
            'affine',
            'subject_id',
        }
        missing_keys = sorted(required_keys - set(data.files))
        if missing_keys:
            raise RuntimeError(f'{subject_id} NPZ missing keys: {missing_keys}')
        if str(data['subject_id'].item()) != subject_id:
            raise RuntimeError(f'{subject_id} NPZ subject identifier mismatch.')

        channel_names = [str(value) for value in data['input_channels'].tolist()]
        if 'ncct_broad_window' not in channel_names:
            raise RuntimeError(f'{subject_id} has no ncct_broad_window channel.')
        broad_index = channel_names.index('ncct_broad_window')

        broad_window = data['image'][broad_index].astype(np.float32)
        crop_mask = data['crop_mask'].astype(bool)
        cow_label = data['cow_label'].astype(np.int16)
        affine = data['affine'].astype(np.float64)

    mask_image = nib.load(str(mask_path))
    mask_array = np.asanyarray(mask_image.dataobj)
    if mask_array.ndim == 4 and mask_array.shape[-1] == 1:
        mask_array = mask_array[..., 0]
    brain_mask = mask_array > 0.5

    return {
        'broad_window': broad_window,
        'crop_mask': crop_mask,
        'cow_label': cow_label,
        'affine': affine,
        'mask_image': mask_image,
        'mask_array': mask_array,
        'brain_mask': brain_mask,
    }


def save_case_qa(
    subject_id,
    broad_window,
    brain_mask,
    cow_label,
    cow_coverage,
    category,
    output_path,
):
    cow_mask = cow_label > 0
    cow_inside = cow_mask & brain_mask
    cow_outside = cow_mask & ~brain_mask
    boundary = mask_boundary(brain_mask)
    selection_mask = cow_outside if cow_outside.any() else cow_mask

    axes = [2, 1, 0]
    names = ['axial', 'coronal', 'sagittal']
    fig, panels = plt.subplots(3, 3, figsize=(15, 13))

    for row, (axis, name) in enumerate(zip(axes, names)):
        index = select_axis_index(selection_mask, axis, brain_mask)
        image_slice = display_slice(broad_window, axis, index)
        boundary_slice = display_slice(boundary, axis, index)
        inside_slice = display_slice(cow_inside, axis, index)
        outside_slice = display_slice(cow_outside, axis, index)
        clipped_slice = display_slice(cow_label * brain_mask, axis, index)

        panels[row, 0].imshow(image_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 0].imshow(
            np.ma.masked_where(~boundary_slice, boundary_slice),
            cmap='autumn',
            alpha=0.95,
        )
        panels[row, 0].set_title(f'{name}: SynthStrip boundary')

        panels[row, 1].imshow(image_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 1].imshow(
            np.ma.masked_where(~inside_slice, inside_slice),
            cmap='Greens',
            alpha=0.90,
        )
        panels[row, 1].imshow(
            np.ma.masked_where(~outside_slice, outside_slice),
            cmap='Reds',
            alpha=0.90,
        )
        panels[row, 1].set_title('CoW: inside green, outside red')

        panels[row, 2].imshow(image_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 2].imshow(
            np.ma.masked_where(clipped_slice == 0, clipped_slice),
            cmap='turbo',
            alpha=0.90,
            interpolation='nearest',
        )
        panels[row, 2].set_title('CoW after intracranial clipping')

        for column in range(3):
            panels[row, column].axis('off')

    fig.suptitle(
        f'{subject_id} CoW compatibility QA\n'
        f'coverage={cow_coverage:.4f}, category={category}',
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


required_paths = [
    FROZEN_MANIFEST_PATH,
    SELECTION_GATE_PATH,
    GENERATION_AUDIT_PATH,
    GENERATION_SUMMARY_PATH,
    CASE_ROOT,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 bulk SynthStrip verification inputs are missing:\n'
        + '\n'.join(missing_paths)
    )

selection_gate = json.loads(SELECTION_GATE_PATH.read_text(encoding='utf-8'))
generation_summary = json.loads(
    GENERATION_SUMMARY_PATH.read_text(encoding='utf-8')
)
if selection_gate.get('gate_passed') is not True:
    raise RuntimeError('Trial8 transform-selection gate did not pass.')
if selection_gate.get('bulk_synthstrip_mask_generation_authorized') is not True:
    raise RuntimeError('Bulk SynthStrip generation was not authorized.')
if generation_summary.get('all_subjects_completed') is not True:
    raise RuntimeError('Bulk SynthStrip generation did not complete all subjects.')
if generation_summary.get('target_arrays_read') is not False:
    raise RuntimeError('Generation summary reports target-array access.')

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
generation_df = pd.read_csv(GENERATION_AUDIT_PATH)

required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen manifest missing columns: {missing_columns}')

for name, frame in [('manifest', manifest_df), ('generation audit', generation_df)]:
    if len(frame) != EXPECTED_SUBJECTS:
        raise RuntimeError(
            f'{name} has {len(frame)} rows; expected {EXPECTED_SUBJECTS}.'
        )
    if frame['subject_id'].duplicated().any():
        raise RuntimeError(f'{name} contains duplicate subjects.')

split_counts = manifest_df['dev_split'].value_counts().to_dict()
if split_counts != EXPECTED_SPLITS:
    raise RuntimeError(f'Unexpected split counts: {split_counts}')

manifest_subjects = set(manifest_df['subject_id'].astype(str))
generation_subjects = set(generation_df['subject_id'].astype(str))
if manifest_subjects != generation_subjects:
    raise RuntimeError('Generation audit subjects do not match the manifest.')

for path in [VERIFICATION_ROOT, QA_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

manifest_sha256 = file_sha256(FROZEN_MANIFEST_PATH)
selection_gate_sha256 = file_sha256(SELECTION_GATE_PATH)
generation_audit_sha256 = file_sha256(GENERATION_AUDIT_PATH)
generation_summary_sha256 = file_sha256(GENERATION_SUMMARY_PATH)

manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
generation_lookup = generation_df.set_index('subject_id')

print('Trial8 bulk SynthStrip verification and CoW diagnostic')
print(f'Frozen subjects: {len(manifest_df)}')
print(f'Split counts: {split_counts}')
print('Study trainable parameters: 0 (preprocessing verification only)')
print('Target arrays read: False')
print('Bulk atlas registration authorized: False')

case_rows = []
cow_rows = []
cow_label_rows = []
error_rows = []

for case_index, row in enumerate(manifest_df.itertuples(index=False), start=1):
    subject_id = str(row.subject_id)
    dev_split = str(row.dev_split)
    npz_path = Path(row.vascufate_v2_npz_path)
    case_dir = CASE_ROOT / subject_id
    mask_path = case_dir / 'ncct_synthstrip_mask.nii.gz'
    complete_path = case_dir / 'complete.json'

    try:
        if not npz_path.exists():
            raise FileNotFoundError(f'NPZ not found: {npz_path}')
        if not mask_path.exists():
            raise FileNotFoundError(f'Mask not found: {mask_path}')
        if not complete_path.exists():
            raise FileNotFoundError(f'Completion record not found: {complete_path}')

        complete = json.loads(complete_path.read_text(encoding='utf-8'))
        if complete.get('subject_id') != subject_id:
            raise RuntimeError('Completion-record subject mismatch.')
        if complete.get('dev_split') != dev_split:
            raise RuntimeError('Completion-record split mismatch.')
        if complete.get('target_arrays_read') is not False:
            raise RuntimeError('Completion record reports target-array access.')
        if complete.get('completed') is not True:
            raise RuntimeError('Completion record is not marked complete.')

        observed_npz_sha256 = file_sha256(npz_path)
        observed_mask_sha256 = file_sha256(mask_path)
        npz_hash_matches = complete.get('npz_sha256') == observed_npz_sha256
        mask_hash_matches = complete.get('mask_sha256') == observed_mask_sha256

        arrays = load_case_arrays(subject_id, npz_path, mask_path)
        broad_window = arrays['broad_window']
        crop_mask = arrays['crop_mask']
        cow_label = arrays['cow_label']
        affine = arrays['affine']
        mask_image = arrays['mask_image']
        mask_array = arrays['mask_array']
        brain_mask = arrays['brain_mask']

        shapes_match = bool(
            broad_window.shape
            == crop_mask.shape
            == cow_label.shape
            == brain_mask.shape
        )
        affines_match = bool(
            np.allclose(mask_image.affine, affine, atol=1e-5, rtol=1e-5)
        )
        orientation = ''.join(nib.aff2axcodes(affine))
        spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
        spacing_matches = bool(
            np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0)
        )
        mask_finite = bool(np.isfinite(mask_array).all())
        mask_binary = bool(
            set(np.unique(mask_array).tolist()).issubset({0, 1, 0.0, 1.0})
        )

        labeled, component_count = ndimage.label(brain_mask)
        component_sizes = np.bincount(labeled.ravel())[1:]
        brain_voxels = int(brain_mask.sum())
        largest_component_fraction = (
            float(component_sizes.max() / brain_voxels)
            if len(component_sizes) and brain_voxels
            else 0.0
        )
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        brain_volume_ml = float(brain_voxels * voxel_volume_ml)
        crop_containment = float(
            np.logical_and(brain_mask, crop_mask).sum()
            / max(brain_voxels, 1)
        )
        edge_touch_x = bool(brain_mask[0].any() or brain_mask[-1].any())
        edge_touch_y = bool(
            brain_mask[:, 0].any() or brain_mask[:, -1].any()
        )
        edge_touch_z = bool(
            brain_mask[:, :, 0].any() or brain_mask[:, :, -1].any()
        )

        structural_checks = {
            'npz_hash_matches_completion_record': bool(npz_hash_matches),
            'mask_hash_matches_completion_record': bool(mask_hash_matches),
            'all_spatial_shapes_match': shapes_match,
            'mask_affine_matches_npz': affines_match,
            'orientation_is_ras': orientation == 'RAS',
            'spacing_is_0p8_0p8_2p0_mm': spacing_matches,
            'mask_is_finite': mask_finite,
            'mask_is_binary': mask_binary,
            'mask_is_nonempty': brain_voxels > 0,
            'brain_volume_between_500_and_2200_ml': bool(
                MIN_BRAIN_VOLUME_ML
                <= brain_volume_ml
                <= MAX_BRAIN_VOLUME_ML
            ),
            'largest_component_fraction_at_least_0p995': bool(
                largest_component_fraction
                >= MIN_LARGEST_COMPONENT_FRACTION
            ),
            'brain_mask_at_least_0p99_inside_crop_mask': bool(
                crop_containment >= MIN_CROP_CONTAINMENT
            ),
            'does_not_touch_x_or_y_crop_edges': bool(
                not (edge_touch_x or edge_touch_y)
            ),
        }
        structural_pass = bool(all(structural_checks.values()))

        generation_row = generation_lookup.loc[subject_id]
        case_rows.append({
            'subject_id': subject_id,
            'dev_split': dev_split,
            'npz_path': str(npz_path),
            'mask_path': str(mask_path),
            'npz_sha256': observed_npz_sha256,
            'mask_sha256': observed_mask_sha256,
            'npz_hash_matches_completion_record': npz_hash_matches,
            'mask_hash_matches_completion_record': mask_hash_matches,
            'all_spatial_shapes_match': shapes_match,
            'mask_affine_matches_npz': affines_match,
            'orientation': orientation,
            'spacing_x_mm': float(spacing[0]),
            'spacing_y_mm': float(spacing[1]),
            'spacing_z_mm': float(spacing[2]),
            'mask_finite': mask_finite,
            'mask_binary': mask_binary,
            'brain_voxels': brain_voxels,
            'brain_volume_ml': brain_volume_ml,
            'component_count': int(component_count),
            'largest_component_fraction': largest_component_fraction,
            'crop_containment_fraction': crop_containment,
            'edge_touch_x': edge_touch_x,
            'edge_touch_y': edge_touch_y,
            'edge_touch_z': edge_touch_z,
            'generation_safety_passed_with_cow_gate': as_bool(
                generation_row['automated_safety_passed']
            ),
            'structural_mask_safety_passed': structural_pass,
            'target_arrays_read': False,
        })

        cow_mask = cow_label > 0
        cow_voxels = int(cow_mask.sum())
        cow_inside = cow_mask & brain_mask
        cow_outside = cow_mask & ~brain_mask
        cow_inside_voxels = int(cow_inside.sum())
        cow_outside_voxels = int(cow_outside.sum())
        cow_coverage = (
            float(cow_inside_voxels / cow_voxels)
            if cow_voxels
            else np.nan
        )

        if cow_outside_voxels:
            distance_to_brain_mm = ndimage.distance_transform_edt(
                ~brain_mask,
                sampling=spacing,
            )
            outside_distances = distance_to_brain_mm[cow_outside]
            outside_distance_median_mm = float(np.median(outside_distances))
            outside_distance_p95_mm = float(np.quantile(outside_distances, 0.95))
            outside_distance_max_mm = float(outside_distances.max())
            outside_within_1mm_fraction = float(
                np.mean(outside_distances <= 1.0)
            )
            outside_within_2mm_fraction = float(
                np.mean(outside_distances <= 2.0)
            )
            outside_within_5mm_fraction = float(
                np.mean(outside_distances <= 5.0)
            )
        else:
            outside_distance_median_mm = 0.0
            outside_distance_p95_mm = 0.0
            outside_distance_max_mm = 0.0
            outside_within_1mm_fraction = 1.0
            outside_within_2mm_fraction = 1.0
            outside_within_5mm_fraction = 1.0

        category = cow_category(
            cow_voxels,
            cow_coverage,
            outside_distance_p95_mm,
        )
        requires_visual_review = bool(
            cow_voxels > 0 and cow_coverage < COW_REVIEW_THRESHOLD
        )

        cow_rows.append({
            'subject_id': subject_id,
            'dev_split': dev_split,
            'cow_voxels': cow_voxels,
            'cow_inside_brain_voxels': cow_inside_voxels,
            'cow_outside_brain_voxels': cow_outside_voxels,
            'cow_coverage_fraction': cow_coverage,
            'outside_distance_median_mm': outside_distance_median_mm,
            'outside_distance_p95_mm': outside_distance_p95_mm,
            'outside_distance_max_mm': outside_distance_max_mm,
            'outside_within_1mm_fraction': outside_within_1mm_fraction,
            'outside_within_2mm_fraction': outside_within_2mm_fraction,
            'outside_within_5mm_fraction': outside_within_5mm_fraction,
            'cow_compatibility_category': category,
            'requires_visual_review': requires_visual_review,
            'structural_mask_safety_passed': structural_pass,
            'target_arrays_read': False,
        })

        for label_value in sorted(
            int(value) for value in np.unique(cow_label) if int(value) > 0
        ):
            label_mask = cow_label == label_value
            label_voxels = int(label_mask.sum())
            label_inside = int(np.logical_and(label_mask, brain_mask).sum())
            cow_label_rows.append({
                'subject_id': subject_id,
                'dev_split': dev_split,
                'cow_label': label_value,
                'label_voxels': label_voxels,
                'inside_brain_voxels': label_inside,
                'outside_brain_voxels': label_voxels - label_inside,
                'inside_brain_fraction': float(label_inside / label_voxels),
                'target_arrays_read': False,
            })

        if case_index == 1 or case_index % 20 == 0 or case_index == EXPECTED_SUBJECTS:
            print(
                f'Verified {case_index}/{EXPECTED_SUBJECTS}: {subject_id}, '
                f'structural={structural_pass}, '
                f'CoW={cow_coverage:.4f}, category={category}'
            )

    except Exception as error:
        error_rows.append({
            'subject_id': subject_id,
            'dev_split': dev_split,
            'error_type': type(error).__name__,
            'error': str(error),
            'target_arrays_read': False,
        })
        print(
            f'FAILED {case_index}/{EXPECTED_SUBJECTS}: {subject_id}: '
            f'{type(error).__name__}: {error}'
        )

case_df = pd.DataFrame(case_rows).sort_values('subject_id').reset_index(drop=True)
cow_df = pd.DataFrame(cow_rows).sort_values('subject_id').reset_index(drop=True)
cow_label_df = pd.DataFrame(cow_label_rows).sort_values(
    ['subject_id', 'cow_label']
).reset_index(drop=True)
error_df = pd.DataFrame(error_rows)

case_df.to_csv(CASE_AUDIT_PATH, index=False)
cow_df.to_csv(COW_AUDIT_PATH, index=False)
cow_label_df.to_csv(COW_LABEL_AUDIT_PATH, index=False)

if error_rows:
    print('\nVerification errors:')
    display(error_df)
    raise RuntimeError(
        'Trial8 bulk mask verification encountered errors. Do not continue.'
    )

all_structural_masks_passed = bool(
    len(case_df) == EXPECTED_SUBJECTS
    and case_df['structural_mask_safety_passed'].map(as_bool).all()
)

qa_reasons = {}


def add_qa_reason(subject_id, reason):
    qa_reasons.setdefault(str(subject_id), set()).add(reason)


for row in case_df.loc[
    ~case_df['structural_mask_safety_passed'].map(as_bool)
].itertuples(index=False):
    add_qa_reason(row.subject_id, 'structural_mask_failure')

for row in cow_df.loc[cow_df['requires_visual_review']].itertuples(index=False):
    add_qa_reason(row.subject_id, 'cow_coverage_below_0p95')

for split_name, split_df in case_df.groupby('dev_split'):
    add_qa_reason(
        split_df.nsmallest(1, 'brain_volume_ml').iloc[0]['subject_id'],
        f'{split_name}_minimum_brain_volume',
    )
    add_qa_reason(
        split_df.nlargest(1, 'brain_volume_ml').iloc[0]['subject_id'],
        f'{split_name}_maximum_brain_volume',
    )
    add_qa_reason(
        split_df.nsmallest(1, 'crop_containment_fraction').iloc[0]['subject_id'],
        f'{split_name}_minimum_crop_containment',
    )

qa_rows = []
manifest_lookup = manifest_df.set_index('subject_id')
cow_lookup = cow_df.set_index('subject_id')

for qa_index, subject_id in enumerate(sorted(qa_reasons), start=1):
    manifest_row = manifest_lookup.loc[subject_id]
    cow_row = cow_lookup.loc[subject_id]
    npz_path = Path(manifest_row['vascufate_v2_npz_path'])
    mask_path = CASE_ROOT / subject_id / 'ncct_synthstrip_mask.nii.gz'
    qa_path = QA_ROOT / f'{subject_id}_synthstrip_cow_qa.png'

    arrays = load_case_arrays(subject_id, npz_path, mask_path)
    save_case_qa(
        subject_id=subject_id,
        broad_window=arrays['broad_window'],
        brain_mask=arrays['brain_mask'],
        cow_label=arrays['cow_label'],
        cow_coverage=float(cow_row['cow_coverage_fraction']),
        category=str(cow_row['cow_compatibility_category']),
        output_path=qa_path,
    )
    qa_rows.append({
        'subject_id': subject_id,
        'dev_split': str(manifest_row['dev_split']),
        'qa_reasons': ';'.join(sorted(qa_reasons[subject_id])),
        'cow_coverage_fraction': float(cow_row['cow_coverage_fraction']),
        'cow_compatibility_category': str(
            cow_row['cow_compatibility_category']
        ),
        'qa_path': str(qa_path),
        'target_arrays_read': False,
    })
    print(f'Saved QA {qa_index}/{len(qa_reasons)}: {subject_id}')

qa_df = pd.DataFrame(qa_rows).sort_values('subject_id').reset_index(drop=True)
qa_df.to_csv(QA_SELECTION_PATH, index=False)

structural_summary = case_df.groupby('dev_split', as_index=False).agg(
    subjects=('subject_id', 'count'),
    structural_passed=('structural_mask_safety_passed', 'sum'),
    minimum_brain_volume_ml=('brain_volume_ml', 'min'),
    maximum_brain_volume_ml=('brain_volume_ml', 'max'),
    minimum_crop_containment=('crop_containment_fraction', 'min'),
    minimum_largest_component_fraction=(
        'largest_component_fraction',
        'min',
    ),
)
cow_category_summary = (
    cow_df.groupby(
        ['dev_split', 'cow_compatibility_category'],
        as_index=False,
    )
    .agg(
        subjects=('subject_id', 'count'),
        minimum_coverage=('cow_coverage_fraction', 'min'),
        median_coverage=('cow_coverage_fraction', 'median'),
        maximum_outside_p95_mm=('outside_distance_p95_mm', 'max'),
    )
    .sort_values(['dev_split', 'cow_compatibility_category'])
)

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'verification_version': VERIFICATION_VERSION,
    'preprocessing_version': PREPROCESSING_VERSION,
    'synthstrip_version': SYNTHSTRIP_VERSION,
    'bulk_version': BULK_VERSION,
    'subjects_verified': int(len(case_df)),
    'all_structural_masks_passed': all_structural_masks_passed,
    'cow_is_a_separate_compatibility_check': True,
    'cow_review_threshold': COW_REVIEW_THRESHOLD,
    'cow_cases_requiring_visual_review': int(
        cow_df['requires_visual_review'].sum()
    ),
    'cow_gross_mismatch_cases': int(
        (cow_df['cow_compatibility_category'] == 'gross_mismatch').sum()
    ),
    'visual_qa_cases': int(len(qa_df)),
    'manual_visual_qa_approved': False,
    'masks_eligible_for_manual_approval': all_structural_masks_passed,
    'bulk_atlas_registration_authorized': False,
    'target_arrays_read': False,
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'frozen_manifest_sha256': manifest_sha256,
    'selection_gate_path': str(SELECTION_GATE_PATH),
    'selection_gate_sha256': selection_gate_sha256,
    'generation_audit_path': str(GENERATION_AUDIT_PATH),
    'generation_audit_sha256': generation_audit_sha256,
    'generation_summary_path': str(GENERATION_SUMMARY_PATH),
    'generation_summary_sha256': generation_summary_sha256,
    'case_audit_path': str(CASE_AUDIT_PATH),
    'cow_audit_path': str(COW_AUDIT_PATH),
    'cow_label_audit_path': str(COW_LABEL_AUDIT_PATH),
    'qa_selection_path': str(QA_SELECTION_PATH),
    'qa_root': str(QA_ROOT),
}
atomic_write_json(summary, SUMMARY_PATH)

print('\nStructural brain-mask verification:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    240,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(structural_summary)

print('\nCoW compatibility categories:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    240,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(cow_category_summary)

print('\nLowest CoW compatibility cases:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    240,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(cow_df.nsmallest(15, 'cow_coverage_fraction'))

print('\n' + '=' * 80)
print('TRIAL8 BULK SYNTHSTRIP VERIFICATION COMPLETE')
print('=' * 80)
print(f'Structural masks passed:              {all_structural_masks_passed}')
print(
    'CoW cases requiring visual review:  '
    f'{int(cow_df["requires_visual_review"].sum())}'
)
print(
    'CoW gross-mismatch cases:            '
    f'{int((cow_df["cow_compatibility_category"] == "gross_mismatch").sum())}'
)
print(f'Visual QA images created:             {len(qa_df)}')
print('Study trainable parameters:           0 (verification only)')
print('Target arrays read:                    False')
print('Bulk atlas registration authorized:   False')
print(f'Structural audit:                      {CASE_AUDIT_PATH}')
print(f'CoW compatibility audit:              {COW_AUDIT_PATH}')
print(f'CoW label audit:                      {COW_LABEL_AUDIT_PATH}')
print(f'Visual QA selection:                  {QA_SELECTION_PATH}')
print(f'Visual QA directory:                  {QA_ROOT}')
print(f'Summary:                              {SUMMARY_PATH}')

if not all_structural_masks_passed:
    failed = case_df.loc[
        ~case_df['structural_mask_safety_passed'].map(as_bool)
    ]
    print('\nStructural mask failures:')
    display(failed)
    raise RuntimeError(
        'One or more SynthStrip masks failed structural verification. '
        'Do not begin atlas registration.'
    )

print('\nStop here and check the visual QA images before approval.')


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
BULK_VERSION = 'bulk_all148_v1'
VERIFICATION_VERSION = 'v1_structural_mask_and_cow_compatibility'

VERIFICATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / BULK_VERSION
    / 'verification'
    / VERIFICATION_VERSION
)
CASE_AUDIT_PATH = VERIFICATION_ROOT / 'mask_structural_audit.csv'
COW_AUDIT_PATH = VERIFICATION_ROOT / 'cow_compatibility_audit.csv'
QA_SELECTION_PATH = VERIFICATION_ROOT / 'visual_qa_selection.csv'
SUMMARY_PATH = VERIFICATION_ROOT / 'verification_summary.json'

EXPECTED_SUBJECTS = 148
EXPECTED_GROSS_MISMATCH_SUBJECT = 'sub-stroke0016'


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == 'true'


required_paths = [
    CASE_AUDIT_PATH,
    COW_AUDIT_PATH,
    QA_SELECTION_PATH,
    SUMMARY_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 bulk visual-review inputs are missing:\n'
        + '\n'.join(missing_paths)
    )

summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
case_df = pd.read_csv(CASE_AUDIT_PATH)
cow_df = pd.read_csv(COW_AUDIT_PATH)
qa_df = pd.read_csv(QA_SELECTION_PATH)

if len(case_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Structural audit has {len(case_df)} rows; expected {EXPECTED_SUBJECTS}.'
    )
if not case_df['structural_mask_safety_passed'].map(as_bool).all():
    raise RuntimeError('At least one structural brain-mask check failed.')
if summary.get('all_structural_masks_passed') is not True:
    raise RuntimeError('Verification summary did not pass structural masks.')
if summary.get('target_arrays_read') is not False:
    raise RuntimeError('Verification summary reports target-array access.')
if summary.get('bulk_atlas_registration_authorized') is not False:
    raise RuntimeError('Bulk atlas registration must remain locked.')

gross_df = cow_df.loc[
    cow_df['cow_compatibility_category'] == 'gross_mismatch'
]
if set(gross_df['subject_id'].astype(str)) != {
    EXPECTED_GROSS_MISMATCH_SUBJECT
}:
    raise RuntimeError(
        'The gross-mismatch subject set changed. Stop and check the audit.'
    )

missing_qa = [
    str(path)
    for path in qa_df['qa_path'].map(Path)
    if not path.exists()
]
if missing_qa:
    raise FileNotFoundError(
        'Visual QA images are missing:\n' + '\n'.join(missing_qa)
    )

review_df = qa_df.merge(
    cow_df[[
        'subject_id',
        'cow_voxels',
        'cow_inside_brain_voxels',
        'cow_outside_brain_voxels',
        'outside_distance_median_mm',
        'outside_distance_p95_mm',
        'outside_distance_max_mm',
        'outside_within_5mm_fraction',
        'requires_visual_review',
    ]],
    on='subject_id',
    how='left',
    validate='one_to_one',
)

category_priority = {
    'gross_mismatch': 0,
    'moderate_mismatch': 1,
    'near_boundary_spill': 2,
    'strong_overlap': 3,
    'cow_absent': 4,
}
review_df['category_priority'] = review_df[
    'cow_compatibility_category'
].map(category_priority).fillna(9)
review_df = review_df.sort_values(
    ['category_priority', 'cow_coverage_fraction', 'subject_id'],
    ascending=[True, True, True],
).reset_index(drop=True)

print('Trial8 bulk SynthStrip visual QA review')
print(f'Structurally valid masks:          {len(case_df)}/{EXPECTED_SUBJECTS}')
print(
    'CoW cases requiring review:     '
    f'{int(cow_df["requires_visual_review"].map(as_bool).sum())}'
)
print(f'Gross CoW mismatches:             {len(gross_df)}')
print(f'QA images to check:               {len(review_df)}')
print('Study trainable parameters:       0 (review only)')
print('Target arrays read:                False')
print('Bulk atlas registration:           not authorized')

print('\nReview order and measurements:')
columns = [
    'subject_id',
    'dev_split',
    'cow_compatibility_category',
    'cow_coverage_fraction',
    'cow_voxels',
    'cow_outside_brain_voxels',
    'outside_distance_p95_mm',
    'outside_distance_max_mm',
    'outside_within_5mm_fraction',
    'qa_reasons',
]
with pd.option_context(
    'display.max_rows',
    None,
    'display.max_columns',
    None,
    'display.width',
    260,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(review_df[columns])

display(Markdown(
    '## Visual review checklist\n\n'
    '- [ ] The orange SynthStrip boundary follows the intracranial boundary.\n'
    '- [ ] No cerebral, cerebellar, or brainstem tissue is truncated.\n'
    '- [ ] Red CoW spill in boundary-spill cases remains immediately adjacent '
    'to the brain boundary.\n'
    '- [ ] Green clipped CoW voxels remain anatomically plausible.\n'
    '- [ ] No left-right or superior-inferior spatial mismatch is visible.\n'
    '- [ ] `sub-stroke0016` shows a detached CoW source rather than a failed '
    'SynthStrip brain mask.\n'
))

for index, row in enumerate(review_df.itertuples(index=False), start=1):
    display(Markdown(
        f'## {index}/{len(review_df)}: {row.subject_id}\n\n'
        f'- Split: `{row.dev_split}`\n'
        f'- Category: `{row.cow_compatibility_category}`\n'
        f'- CoW coverage: `{row.cow_coverage_fraction:.6f}`\n'
        f'- CoW voxels inside/outside: '
        f'`{int(row.cow_inside_brain_voxels)} / '
        f'{int(row.cow_outside_brain_voxels)}`\n'
        f'- Outside distance median/p95/max: '
        f'`{row.outside_distance_median_mm:.3f} / '
        f'{row.outside_distance_p95_mm:.3f} / '
        f'{row.outside_distance_max_mm:.3f} mm`\n'
        f'- QA reasons: `{row.qa_reasons}`\n'
    ))
    display(Image(filename=str(row.qa_path), width=1100))

print('\n' + '=' * 80)
print('TRIAL8 BULK SYNTHSTRIP VISUAL QA DISPLAY COMPLETE')
print('=' * 80)
print('Approval state changed:              False')
print('Bulk atlas registration authorized: False')
print('Target arrays read:                  False')
print('\nCheck all displayed images before approving the mask and CoW policy.')


In [ ]:
import json
import hashlib
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage
from nibabel.processing import resample_from_to
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
BULK_VERSION = 'bulk_all148_v1'
SUBJECT_ID = 'sub-stroke0016'
DIAGNOSTIC_VERSION = 'v1_source_vs_preprocessed_cow_geometry'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
MASK_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / BULK_VERSION
    / 'cases'
    / SUBJECT_ID
    / 'ncct_synthstrip_mask.nii.gz'
)
VERIFICATION_SUMMARY_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / BULK_VERSION
    / 'verification'
    / 'v1_structural_mask_and_cow_compatibility'
    / 'verification_summary.json'
)
DIAGNOSTIC_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / BULK_VERSION
    / 'diagnostics'
    / SUBJECT_ID
    / DIAGNOSTIC_VERSION
)
GRID_AUDIT_PATH = DIAGNOSTIC_ROOT / 'source_grid_audit.csv'
OVERLAP_AUDIT_PATH = DIAGNOSTIC_ROOT / 'source_vs_preprocessed_overlap.csv'
CROP_AUDIT_PATH = DIAGNOSTIC_ROOT / 'crop_influence_audit.csv'
QA_PATH = DIAGNOSTIC_ROOT / 'source_vs_preprocessed_cow_qa.png'
SUMMARY_PATH = DIAGNOSTIC_ROOT / 'diagnostic_summary.json'

EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)
DOCUMENTED_COW_LABELS = set(range(1, 14))


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def atomic_write_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)


def effective_3d_image(path):
    source = nib.load(str(path))
    array = np.asanyarray(source.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 3:
        raise RuntimeError(f'{path} has unsupported shape {array.shape}.')
    image = nib.Nifti1Image(array, source.affine, source.header)
    image.set_sform(source.affine, code=int(source.header['sform_code']))
    image.set_qform(source.affine, code=int(source.header['qform_code']))
    return source, image


def canonical_3d_image(path):
    raw, effective = effective_3d_image(path)
    canonical = nib.as_closest_canonical(effective)
    if ''.join(nib.aff2axcodes(canonical.affine)) != 'RAS':
        raise RuntimeError(f'Could not canonicalize {path} to RAS.')
    return raw, effective, canonical


def spatial_corners_world(shape, affine):
    corners = np.asarray(list(product(*[(0, size - 1) for size in shape])))
    return nib.affines.apply_affine(affine, corners)


def world_bbox(shape, affine):
    corners = spatial_corners_world(shape, affine)
    return corners.min(axis=0), corners.max(axis=0)


def mask_bbox(mask):
    coordinates = np.argwhere(mask)
    if not len(coordinates):
        return None, None
    return coordinates.min(axis=0), coordinates.max(axis=0)


def mask_centroid_world(mask, affine):
    coordinates = np.argwhere(mask)
    if not len(coordinates):
        return np.full(3, np.nan, dtype=np.float64)
    centroid_voxel = coordinates.mean(axis=0)
    return np.asarray(
        nib.affines.apply_affine(affine, centroid_voxel),
        dtype=np.float64,
    )


def overlap_metrics(name, mask, brain_mask, affine):
    mask = np.asarray(mask, dtype=bool)
    brain_mask = np.asarray(brain_mask, dtype=bool)
    spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
    total = int(mask.sum())
    inside = int(np.logical_and(mask, brain_mask).sum())
    outside_mask = mask & ~brain_mask
    outside = int(outside_mask.sum())

    if outside:
        distance = ndimage.distance_transform_edt(
            ~brain_mask,
            sampling=spacing,
        )[outside_mask]
        distance_median = float(np.median(distance))
        distance_p95 = float(np.quantile(distance, 0.95))
        distance_max = float(distance.max())
    else:
        distance_median = 0.0
        distance_p95 = 0.0
        distance_max = 0.0

    center = mask_centroid_world(mask, affine)
    brain_center = mask_centroid_world(brain_mask, affine)
    center_distance = (
        float(np.linalg.norm(center - brain_center))
        if np.isfinite(center).all() and np.isfinite(brain_center).all()
        else np.nan
    )
    minimum, maximum = mask_bbox(mask)

    return {
        'representation': name,
        'voxels': total,
        'inside_brain_voxels': inside,
        'outside_brain_voxels': outside,
        'inside_brain_fraction': float(inside / total) if total else np.nan,
        'outside_distance_median_mm': distance_median,
        'outside_distance_p95_mm': distance_p95,
        'outside_distance_max_mm': distance_max,
        'centroid_world_x': float(center[0]),
        'centroid_world_y': float(center[1]),
        'centroid_world_z': float(center[2]),
        'centroid_to_brain_center_mm': center_distance,
        'bbox_min_i': None if minimum is None else int(minimum[0]),
        'bbox_min_j': None if minimum is None else int(minimum[1]),
        'bbox_min_k': None if minimum is None else int(minimum[2]),
        'bbox_max_i': None if maximum is None else int(maximum[0]),
        'bbox_max_j': None if maximum is None else int(maximum[1]),
        'bbox_max_k': None if maximum is None else int(maximum[2]),
    }


def display_slice(array, axis, index):
    return np.rot90(np.take(array, int(index), axis=axis))


def mask_boundary(mask):
    mask = np.asarray(mask, dtype=bool)
    return mask & ~ndimage.binary_erosion(mask, iterations=1)


def axis_index(mask, axis, fallback):
    reduce_axes = tuple(index for index in range(3) if index != axis)
    counts = np.asarray(mask, dtype=np.int64).sum(axis=reduce_axes)
    if counts.max() > 0:
        return int(np.argmax(counts))
    coordinates = np.argwhere(fallback)
    if len(coordinates):
        return int(np.rint(np.median(coordinates[:, axis])))
    return int(fallback.shape[axis] // 2)


def normalize_for_display(array, low_percentile=1.0, high_percentile=99.0):
    array = np.asarray(array, dtype=np.float32)
    finite = array[np.isfinite(array)]
    if not len(finite):
        return np.zeros_like(array, dtype=np.float32)
    low = float(np.percentile(finite, low_percentile))
    high = float(np.percentile(finite, high_percentile))
    if high <= low:
        return np.zeros_like(array, dtype=np.float32)
    return np.clip((array - low) / (high - low), 0.0, 1.0)


def largest_component_overlapping(candidate, preferred_mask):
    labels, count = ndimage.label(candidate)
    if count == 0:
        return np.zeros_like(candidate, dtype=bool)
    preferred_counts = np.bincount(
        labels[preferred_mask].ravel(),
        minlength=count + 1,
    )
    preferred_counts[0] = 0
    if preferred_counts.max() > 0:
        selected = int(np.argmax(preferred_counts))
    else:
        sizes = np.bincount(labels.ravel(), minlength=count + 1)
        sizes[0] = 0
        selected = int(np.argmax(sizes))
    return labels == selected


def reconstruct_crop_support_without_cow(
    ncct_hu,
    perfusion_valid,
    lvo_proximity,
):
    finite = np.isfinite(ncct_hu)
    head_candidate = (
        finite
        & (ncct_hu > -500.0)
        & (ncct_hu < 3000.0)
    )
    structure = ndimage.generate_binary_structure(3, 1)
    head_candidate = ndimage.binary_closing(
        head_candidate,
        structure=structure,
        iterations=3,
    )
    for z_index in range(head_candidate.shape[2]):
        head_candidate[:, :, z_index] = ndimage.binary_fill_holes(
            head_candidate[:, :, z_index]
        )
    head_mask = largest_component_overlapping(
        head_candidate,
        preferred_mask=perfusion_valid,
    )
    head_mask = ndimage.binary_dilation(
        head_mask,
        structure=structure,
        iterations=2,
    )
    support = head_mask | perfusion_valid | (lvo_proximity > 0.01)
    return ndimage.binary_closing(
        support,
        structure=structure,
        iterations=2,
    )


def bbox_with_margin(mask, spacing, margin_mm=(16.0, 16.0, 10.0)):
    coordinates = np.argwhere(mask)
    if not len(coordinates):
        return None, None
    minimum = coordinates.min(axis=0)
    maximum = coordinates.max(axis=0) + 1
    margin = np.ceil(
        np.asarray(margin_mm) / np.asarray(spacing)
    ).astype(int)
    minimum = np.maximum(minimum - margin, 0)
    maximum = np.minimum(maximum + margin, np.asarray(mask.shape))
    return minimum, maximum


def save_qa(
    source_ncct,
    source_cta,
    source_brain,
    source_cow,
    preprocessed_ncct,
    preprocessed_brain,
    preprocessed_cow,
    output_path,
):
    source_ncct_display = np.clip((source_ncct + 20.0) / 120.0, 0.0, 1.0)
    source_cta_display = normalize_for_display(source_cta, 1.0, 99.5)
    preprocessed_display = np.asarray(preprocessed_ncct, dtype=np.float32)

    source_boundary = mask_boundary(source_brain)
    preprocessed_boundary = mask_boundary(preprocessed_brain)
    source_inside = source_cow & source_brain
    source_outside = source_cow & ~source_brain
    preprocessed_inside = preprocessed_cow & preprocessed_brain
    preprocessed_outside = preprocessed_cow & ~preprocessed_brain

    axes = [2, 1, 0]
    names = ['axial', 'coronal', 'sagittal']
    fig, panels = plt.subplots(3, 4, figsize=(19, 13))

    for row, (axis, name) in enumerate(zip(axes, names)):
        source_index = axis_index(source_cow, axis, source_brain)
        preprocessed_index = axis_index(
            preprocessed_cow,
            axis,
            preprocessed_brain,
        )

        ncct_slice = display_slice(source_ncct_display, axis, source_index)
        cta_slice = display_slice(source_cta_display, axis, source_index)
        source_boundary_slice = display_slice(
            source_boundary,
            axis,
            source_index,
        )
        source_inside_slice = display_slice(source_inside, axis, source_index)
        source_outside_slice = display_slice(source_outside, axis, source_index)

        preprocessed_slice = display_slice(
            preprocessed_display,
            axis,
            preprocessed_index,
        )
        preprocessed_boundary_slice = display_slice(
            preprocessed_boundary,
            axis,
            preprocessed_index,
        )
        preprocessed_inside_slice = display_slice(
            preprocessed_inside,
            axis,
            preprocessed_index,
        )
        preprocessed_outside_slice = display_slice(
            preprocessed_outside,
            axis,
            preprocessed_index,
        )

        panels[row, 0].imshow(ncct_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 0].imshow(
            np.ma.masked_where(~source_boundary_slice, source_boundary_slice),
            cmap='autumn',
            alpha=0.95,
        )
        panels[row, 0].set_title(f'{name}: source NCCT + brain')

        panels[row, 1].imshow(ncct_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 1].imshow(
            np.ma.masked_where(~source_inside_slice, source_inside_slice),
            cmap='Greens',
            alpha=0.90,
        )
        panels[row, 1].imshow(
            np.ma.masked_where(~source_outside_slice, source_outside_slice),
            cmap='Reds',
            alpha=0.90,
        )
        panels[row, 1].set_title('source CoW: inside green, outside red')

        panels[row, 2].imshow(cta_slice, cmap='gray', vmin=0, vmax=1)
        panels[row, 2].imshow(
            np.ma.masked_where(~source_outside_slice, source_outside_slice),
            cmap='Reds',
            alpha=0.90,
        )
        panels[row, 2].set_title('source CTA + outside CoW')

        panels[row, 3].imshow(
            preprocessed_slice,
            cmap='gray',
            vmin=0,
            vmax=1,
        )
        panels[row, 3].imshow(
            np.ma.masked_where(
                ~preprocessed_boundary_slice,
                preprocessed_boundary_slice,
            ),
            cmap='autumn',
            alpha=0.90,
        )
        panels[row, 3].imshow(
            np.ma.masked_where(
                ~preprocessed_inside_slice,
                preprocessed_inside_slice,
            ),
            cmap='Greens',
            alpha=0.90,
        )
        panels[row, 3].imshow(
            np.ma.masked_where(
                ~preprocessed_outside_slice,
                preprocessed_outside_slice,
            ),
            cmap='Reds',
            alpha=0.90,
        )
        panels[row, 3].set_title('preprocessed NCCT + brain + CoW')

        for column in range(4):
            panels[row, column].axis('off')

    fig.suptitle(
        f'{SUBJECT_ID}: source versus preprocessed CoW geometry',
        fontsize=15,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


required_paths = [
    FROZEN_MANIFEST_PATH,
    MASK_PATH,
    VERIFICATION_SUMMARY_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 sub-stroke0016 diagnostic inputs are missing:\n'
        + '\n'.join(missing_paths)
    )

verification_summary = json.loads(
    VERIFICATION_SUMMARY_PATH.read_text(encoding='utf-8')
)
if verification_summary.get('all_structural_masks_passed') is not True:
    raise RuntimeError('Bulk SynthStrip structural verification did not pass.')
if verification_summary.get('target_arrays_read') is not False:
    raise RuntimeError('Prior verification reports target-array access.')

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
subject_rows = manifest_df.loc[
    manifest_df['subject_id'].astype(str) == SUBJECT_ID
]
if len(subject_rows) != 1:
    raise RuntimeError(f'Expected one manifest row for {SUBJECT_ID}.')
row = subject_rows.iloc[0]

required_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'path_raw_ncct',
    'path_deriv_cta',
    'path_cow_mask',
    'path_lvo_mask',
}
missing_columns = sorted(required_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen manifest missing columns: {missing_columns}')
if str(row['dev_split']) != 'train':
    raise RuntimeError(f'{SUBJECT_ID} is not a training subject.')

source_paths = {
    'ncct': Path(row['path_raw_ncct']),
    'cta': Path(row['path_deriv_cta']),
    'cow': Path(row['path_cow_mask']),
    'lvo': Path(row['path_lvo_mask']),
}
npz_path = Path(row['vascufate_v2_npz_path'])
for role, path in {**source_paths, 'npz': npz_path}.items():
    if not path.exists():
        raise FileNotFoundError(f'{role} input not found: {path}')

assert_project_path_is_safe(DIAGNOSTIC_ROOT)
DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)

source_images = {}
grid_rows = []
for role, path in source_paths.items():
    raw, effective, canonical = canonical_3d_image(path)
    source_images[role] = canonical
    bbox_min, bbox_max = world_bbox(canonical.shape, canonical.affine)
    grid_rows.append({
        'role': role,
        'path': str(path),
        'sha256': file_sha256(path),
        'raw_shape': str(tuple(raw.shape)),
        'effective_shape': str(tuple(effective.shape)),
        'canonical_shape': str(tuple(canonical.shape)),
        'raw_orientation': ''.join(nib.aff2axcodes(raw.affine)),
        'canonical_orientation': ''.join(
            nib.aff2axcodes(canonical.affine)
        ),
        'spacing_x_mm': float(canonical.header.get_zooms()[0]),
        'spacing_y_mm': float(canonical.header.get_zooms()[1]),
        'spacing_z_mm': float(canonical.header.get_zooms()[2]),
        'qform_code': int(raw.header['qform_code']),
        'sform_code': int(raw.header['sform_code']),
        'world_min_x': float(bbox_min[0]),
        'world_min_y': float(bbox_min[1]),
        'world_min_z': float(bbox_min[2]),
        'world_max_x': float(bbox_max[0]),
        'world_max_y': float(bbox_max[1]),
        'world_max_z': float(bbox_max[2]),
    })

grid_df = pd.DataFrame(grid_rows)
grid_df.to_csv(GRID_AUDIT_PATH, index=False)

ncct_image = source_images['ncct']
cta_on_ncct = resample_from_to(
    source_images['cta'],
    (ncct_image.shape, ncct_image.affine),
    order=1,
    mode='constant',
    cval=-1000.0,
)
cow_on_ncct = resample_from_to(
    source_images['cow'],
    (ncct_image.shape, ncct_image.affine),
    order=0,
    mode='constant',
    cval=0.0,
)
lvo_on_ncct = resample_from_to(
    source_images['lvo'],
    (ncct_image.shape, ncct_image.affine),
    order=0,
    mode='constant',
    cval=0.0,
)

source_ncct = np.asarray(ncct_image.dataobj, dtype=np.float32)
source_cta = np.asarray(cta_on_ncct.dataobj, dtype=np.float32)
source_cow_labels = np.rint(
    np.asarray(cow_on_ncct.dataobj, dtype=np.float32)
).astype(np.int16)
source_cow_documented = np.isin(
    source_cow_labels,
    sorted(DOCUMENTED_COW_LABELS),
)
source_cow_union = source_cow_labels > 0
source_lvo = np.asarray(lvo_on_ncct.dataobj, dtype=np.float32) > 0.5

with np.load(npz_path, allow_pickle=False) as data:
    required_keys = {
        'image',
        'input_channels',
        'crop_mask',
        'perfusion_valid',
        'cow_label',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
        'vessel_features_raw',
        'vessel_presence',
        'affine',
        'full_resampled_affine',
        'full_resampled_shape',
        'crop_origin',
        'subject_id',
    }
    missing_keys = sorted(required_keys - set(data.files))
    if missing_keys:
        raise RuntimeError(f'Preprocessed NPZ missing keys: {missing_keys}')
    if str(data['subject_id'].item()) != SUBJECT_ID:
        raise RuntimeError('Preprocessed NPZ subject identifier mismatch.')

    channel_names = [str(value) for value in data['input_channels'].tolist()]
    broad_index = channel_names.index('ncct_broad_window')
    preprocessed_ncct = data['image'][broad_index].astype(np.float32)
    crop_mask = data['crop_mask'].astype(bool)
    perfusion_valid = data['perfusion_valid'].astype(bool)
    preprocessed_cow_label = data['cow_label'].astype(np.int16)
    preprocessed_cow_soft = data['cow_soft_union'].astype(np.float32)
    preprocessed_lvo_proximity = data['lvo_proximity'].astype(np.float32)
    lvo_present = bool(data['lvo_present'].item())
    vessel_features_raw = data['vessel_features_raw'].astype(np.float32)
    vessel_presence = data['vessel_presence'].astype(bool)
    preprocessed_affine = data['affine'].astype(np.float64)
    full_resampled_affine = data['full_resampled_affine'].astype(np.float64)
    full_resampled_shape = data['full_resampled_shape'].astype(np.int64)
    crop_origin = data['crop_origin'].astype(np.int64)

mask_image = nib.load(str(MASK_PATH))
mask_array = np.asanyarray(mask_image.dataobj)
if mask_array.ndim == 4 and mask_array.shape[-1] == 1:
    mask_array = mask_array[..., 0]
preprocessed_brain = mask_array > 0.5

if preprocessed_brain.shape != preprocessed_ncct.shape:
    raise RuntimeError('SynthStrip mask and preprocessed image shapes differ.')
if not np.allclose(
    mask_image.affine,
    preprocessed_affine,
    atol=1e-5,
    rtol=1e-5,
):
    raise RuntimeError('SynthStrip mask and NPZ affines differ.')

brain_on_source_image = resample_from_to(
    nib.Nifti1Image(
        preprocessed_brain.astype(np.uint8),
        preprocessed_affine,
    ),
    (ncct_image.shape, ncct_image.affine),
    order=0,
    mode='constant',
    cval=0.0,
)
source_brain = np.asarray(brain_on_source_image.dataobj) > 0.5

ncct_on_preprocessed_image = resample_from_to(
    ncct_image,
    (preprocessed_ncct.shape, preprocessed_affine),
    order=1,
    mode='constant',
    cval=-1000.0,
)
ncct_hu_on_preprocessed = np.asarray(
    ncct_on_preprocessed_image.dataobj,
    dtype=np.float32,
)

preprocessed_cow = preprocessed_cow_label > 0
preprocessed_cow_union = preprocessed_cow_soft > 0.5
preprocessed_lvo_core = preprocessed_lvo_proximity >= 0.999

overlap_rows = [
    overlap_metrics(
        'source_cow_documented_on_ncct',
        source_cow_documented,
        source_brain,
        ncct_image.affine,
    ),
    overlap_metrics(
        'source_cow_all_positive_on_ncct',
        source_cow_union,
        source_brain,
        ncct_image.affine,
    ),
    overlap_metrics(
        'source_lvo_on_ncct',
        source_lvo,
        source_brain,
        ncct_image.affine,
    ),
    overlap_metrics(
        'preprocessed_cow_documented',
        preprocessed_cow,
        preprocessed_brain,
        preprocessed_affine,
    ),
    overlap_metrics(
        'preprocessed_cow_soft_union_gt_0p5',
        preprocessed_cow_union,
        preprocessed_brain,
        preprocessed_affine,
    ),
    overlap_metrics(
        'preprocessed_lvo_proximity_ge_0p999',
        preprocessed_lvo_core,
        preprocessed_brain,
        preprocessed_affine,
    ),
]
overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(OVERLAP_AUDIT_PATH, index=False)

source_cow_center = mask_centroid_world(
    source_cow_documented,
    ncct_image.affine,
)
preprocessed_cow_center = mask_centroid_world(
    preprocessed_cow,
    preprocessed_affine,
)
cow_centroid_preservation_error_mm = (
    float(np.linalg.norm(source_cow_center - preprocessed_cow_center))
    if np.isfinite(source_cow_center).all()
    and np.isfinite(preprocessed_cow_center).all()
    else np.nan
)

brain_bbox_min, brain_bbox_max = mask_bbox(preprocessed_brain)
cow_bbox_min, cow_bbox_max = mask_bbox(preprocessed_cow_union)
crop_bbox_min, crop_bbox_max = mask_bbox(crop_mask)

crop_support_without_cow = reconstruct_crop_support_without_cow(
    ncct_hu=ncct_hu_on_preprocessed,
    perfusion_valid=perfusion_valid,
    lvo_proximity=preprocessed_lvo_proximity,
)
without_cow_min, without_cow_max = bbox_with_margin(
    crop_support_without_cow,
    spacing=EXPECTED_SPACING,
)

crop_rows = [
    {
        'support': 'saved_crop_mask',
        'voxels': int(crop_mask.sum()),
        'bbox_min': str(None if crop_bbox_min is None else crop_bbox_min.tolist()),
        'bbox_max': str(None if crop_bbox_max is None else crop_bbox_max.tolist()),
    },
    {
        'support': 'synthstrip_brain',
        'voxels': int(preprocessed_brain.sum()),
        'bbox_min': str(None if brain_bbox_min is None else brain_bbox_min.tolist()),
        'bbox_max': str(None if brain_bbox_max is None else brain_bbox_max.tolist()),
    },
    {
        'support': 'cow_soft_union_gt_0p5',
        'voxels': int(preprocessed_cow_union.sum()),
        'bbox_min': str(None if cow_bbox_min is None else cow_bbox_min.tolist()),
        'bbox_max': str(None if cow_bbox_max is None else cow_bbox_max.tolist()),
    },
    {
        'support': 'reconstructed_crop_support_without_cow',
        'voxels': int(crop_support_without_cow.sum()),
        'bbox_min': str(
            None if without_cow_min is None else without_cow_min.tolist()
        ),
        'bbox_max': str(
            None if without_cow_max is None else without_cow_max.tolist()
        ),
    },
]
crop_df = pd.DataFrame(crop_rows)
crop_df.to_csv(CROP_AUDIT_PATH, index=False)

source_cow_metrics = overlap_df.loc[
    overlap_df['representation'] == 'source_cow_documented_on_ncct'
].iloc[0]
preprocessed_cow_metrics = overlap_df.loc[
    overlap_df['representation'] == 'preprocessed_cow_documented'
].iloc[0]

source_cow_detached = bool(
    source_cow_metrics['voxels'] > 0
    and source_cow_metrics['inside_brain_fraction'] < 0.10
    and source_cow_metrics['outside_distance_p95_mm'] > 20.0
)
preprocessed_cow_detached = bool(
    preprocessed_cow_metrics['voxels'] > 0
    and preprocessed_cow_metrics['inside_brain_fraction'] < 0.10
    and preprocessed_cow_metrics['outside_distance_p95_mm'] > 20.0
)
world_location_preserved = bool(
    np.isfinite(cow_centroid_preservation_error_mm)
    and cow_centroid_preservation_error_mm <= 2.5
)

if source_cow_detached and preprocessed_cow_detached and world_location_preserved:
    likely_origin = 'source_cow_geometry_or_content_defect'
elif not source_cow_detached and preprocessed_cow_detached:
    likely_origin = 'preprocessing_transform_error'
else:
    likely_origin = 'indeterminate_requires_manual_review'

crop_influence_detected = bool(
    without_cow_min is not None
    and without_cow_max is not None
    and (
        np.any(without_cow_min > 0)
        or np.any(without_cow_max < np.asarray(preprocessed_ncct.shape))
    )
)

save_qa(
    source_ncct=source_ncct,
    source_cta=source_cta,
    source_brain=source_brain,
    source_cow=source_cow_documented,
    preprocessed_ncct=preprocessed_ncct,
    preprocessed_brain=preprocessed_brain,
    preprocessed_cow=preprocessed_cow,
    output_path=QA_PATH,
)

diagnostic_checks = {
    'subject_is_train_only': str(row['dev_split']) == 'train',
    'preprocessed_spacing_matches_0p8_0p8_2p0': bool(
        np.allclose(
            np.sqrt(np.sum(preprocessed_affine[:3, :3] ** 2, axis=0)),
            EXPECTED_SPACING,
            atol=1e-4,
            rtol=0,
        )
    ),
    'source_cow_is_detached': source_cow_detached,
    'preprocessed_cow_is_detached': preprocessed_cow_detached,
    'cow_world_location_preserved_by_preprocessing': world_location_preserved,
    'cow_influenced_saved_crop_extent': crop_influence_detected,
    'synthstrip_mask_remains_structurally_valid': True,
    'target_arrays_read': False,
}

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'diagnostic_version': DIAGNOSTIC_VERSION,
    'subject_id': SUBJECT_ID,
    'dev_split': str(row['dev_split']),
    'likely_failure_origin': likely_origin,
    'source_cow_detached': source_cow_detached,
    'preprocessed_cow_detached': preprocessed_cow_detached,
    'cow_centroid_preservation_error_mm': (
        cow_centroid_preservation_error_mm
    ),
    'crop_influence_detected': crop_influence_detected,
    'lvo_present_recorded': lvo_present,
    'vessel_nodes_marked_present': int(vessel_presence.sum()),
    'vessel_features_all_finite': bool(np.isfinite(vessel_features_raw).all()),
    'recommended_interim_action': (
        'quarantine CoW-derived crop and vessel features for sub-stroke0016; '
        'do not alter or invent vascular anatomy before manual review'
    ),
    'target_arrays_read': False,
    'bulk_atlas_registration_authorized': False,
    'diagnostic_checks': diagnostic_checks,
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'frozen_manifest_sha256': file_sha256(FROZEN_MANIFEST_PATH),
    'npz_path': str(npz_path),
    'npz_sha256': file_sha256(npz_path),
    'mask_path': str(MASK_PATH),
    'mask_sha256': file_sha256(MASK_PATH),
    'grid_audit_path': str(GRID_AUDIT_PATH),
    'overlap_audit_path': str(OVERLAP_AUDIT_PATH),
    'crop_audit_path': str(CROP_AUDIT_PATH),
    'qa_path': str(QA_PATH),
}
atomic_write_json(summary, SUMMARY_PATH)

print(f'Trial8 focused geometry diagnostic: {SUBJECT_ID}')
print('Study trainable parameters: 0 (diagnostic only)')
print('Target arrays read: False')
print('Bulk atlas registration authorized: False')

print('\nSource-grid audit:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    280,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(grid_df)

print('\nSource versus preprocessed overlap:')
with pd.option_context(
    'display.max_columns',
    None,
    'display.width',
    280,
    'display.float_format',
    lambda value: f'{value:.6f}',
):
    display(overlap_df)

print('\nCrop influence audit:')
display(crop_df)

print('\nDiagnostic checks:')
display(pd.DataFrame([
    {'check': key, 'value': value}
    for key, value in diagnostic_checks.items()
]))

print('\n' + '=' * 80)
print('TRIAL8 SUB-STROKE0016 COW GEOMETRY DIAGNOSTIC COMPLETE')
print('=' * 80)
print(f'Likely failure origin:              {likely_origin}')
print(
    'CoW centroid preservation error:  '
    f'{cow_centroid_preservation_error_mm:.3f} mm'
)
print(f'CoW influenced crop extent:         {crop_influence_detected}')
print(f'LVO present in NPZ:                 {lvo_present}')
print(f'Vessel nodes marked present:        {int(vessel_presence.sum())}')
print('Data changed:                       False')
print('Target arrays read:                 False')
print('Bulk atlas registration:            not authorized')
print(f'Grid audit:                          {GRID_AUDIT_PATH}')
print(f'Overlap audit:                       {OVERLAP_AUDIT_PATH}')
print(f'Crop audit:                          {CROP_AUDIT_PATH}')
print(f'Visual QA:                           {QA_PATH}')
print(f'Summary:                             {SUMMARY_PATH}')
print('\nStop here and check the diagnostic before correcting this case.')


In [ ]:
import os
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage
from nibabel.processing import resample_from_to
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PARENT_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
CORRECTED_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
CORRECTION_VERSION = 'r2_sub0016_invalid_cow_quarantine_v1'
SUBJECT_ID = 'sub-stroke0016'
TARGET_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)
CROP_MARGIN_MM = np.asarray([16.0, 16.0, 10.0], dtype=np.float64)

PARENT_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
DIAGNOSTIC_SUMMARY_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / PARENT_PREPROCESSING_VERSION
    / 'brain_masks'
    / 'synthstrip_v1_default_csf'
    / 'bulk_all148_v1'
    / 'diagnostics'
    / SUBJECT_ID
    / 'v1_source_vs_preprocessed_cow_geometry'
    / 'diagnostic_summary.json'
)
CORRECTION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / CORRECTED_PREPROCESSING_VERSION
)
SUBJECT_OUTPUT_DIR = CORRECTION_ROOT / 'subjects_npz'
CORRECTED_NPZ_PATH = (
    SUBJECT_OUTPUT_DIR
    / f'{SUBJECT_ID}_{CORRECTED_PREPROCESSING_VERSION}.npz'
)
CORRECTION_AUDIT_PATH = CORRECTION_ROOT / 'sub0016_correction_audit.json'
CANDIDATE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_candidate.csv'
)
CONFIG_PATH = (
    PROJECT_ROOT / 'configs' / f'{CORRECTED_PREPROCESSING_VERSION}_config.json'
)

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def atomic_write_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)


def effective_canonical_image(path):
    source = nib.load(str(path))
    array = np.asanyarray(source.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 3:
        raise RuntimeError(f'{path} has unsupported shape {array.shape}.')
    effective = nib.Nifti1Image(array, source.affine)
    canonical = nib.as_closest_canonical(effective)
    if ''.join(nib.aff2axcodes(canonical.affine)) != 'RAS':
        raise RuntimeError(f'Could not canonicalize {path} to RAS.')
    return canonical


def largest_component_overlapping(candidate, preferred_mask):
    labels, count = ndimage.label(candidate)
    if count == 0:
        return np.zeros_like(candidate, dtype=bool)
    preferred_counts = np.bincount(
        labels[preferred_mask].ravel(),
        minlength=count + 1,
    )
    preferred_counts[0] = 0
    if preferred_counts.max() > 0:
        selected = int(np.argmax(preferred_counts))
    else:
        sizes = np.bincount(labels.ravel(), minlength=count + 1)
        sizes[0] = 0
        selected = int(np.argmax(sizes))
    return labels == selected


def construct_crop_support_without_cow(
    ncct_hu,
    perfusion_valid,
    lvo_proximity,
):
    finite = np.isfinite(ncct_hu)
    head_candidate = (
        finite
        & (ncct_hu > -500.0)
        & (ncct_hu < 3000.0)
    )
    structure = ndimage.generate_binary_structure(3, 1)
    head_candidate = ndimage.binary_closing(
        head_candidate,
        structure=structure,
        iterations=3,
    )
    for z_index in range(head_candidate.shape[2]):
        head_candidate[:, :, z_index] = ndimage.binary_fill_holes(
            head_candidate[:, :, z_index]
        )
    head_mask = largest_component_overlapping(
        head_candidate,
        preferred_mask=perfusion_valid,
    )
    head_mask = ndimage.binary_dilation(
        head_mask,
        structure=structure,
        iterations=2,
    )
    crop_support = head_mask | perfusion_valid | (lvo_proximity > 0.01)
    return ndimage.binary_closing(
        crop_support,
        structure=structure,
        iterations=2,
    ).astype(bool)


def bounding_slices_with_margin(mask, spacing, margin_mm):
    coordinates = np.argwhere(mask)
    if not len(coordinates):
        raise RuntimeError('Corrected crop support is empty.')
    minimum = coordinates.min(axis=0)
    maximum = coordinates.max(axis=0) + 1
    margin = np.ceil(margin_mm / spacing).astype(int)
    minimum = np.maximum(minimum - margin, 0)
    maximum = np.minimum(maximum + margin, np.asarray(mask.shape))
    slices = tuple(
        slice(int(start), int(stop))
        for start, stop in zip(minimum, maximum)
    )
    return slices, minimum.astype(np.int64), maximum.astype(np.int64)


def translated_affine(affine, voxel_offset):
    transform = np.eye(4, dtype=np.float64)
    transform[:3, 3] = np.asarray(voxel_offset, dtype=np.float64)
    return np.asarray(affine, dtype=np.float64) @ transform


def crop_array(array, spatial_shape, crop_slices):
    array = np.asarray(array)
    if tuple(array.shape) == tuple(spatial_shape):
        return array[crop_slices]
    if array.ndim == 4 and tuple(array.shape[1:]) == tuple(spatial_shape):
        return array[(slice(None), *crop_slices)]
    return array.copy()


required_paths = [PARENT_MANIFEST_PATH, DIAGNOSTIC_SUMMARY_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 r2 correction prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

diagnostic = json.loads(DIAGNOSTIC_SUMMARY_PATH.read_text(encoding='utf-8'))
if diagnostic.get('subject_id') != SUBJECT_ID:
    raise RuntimeError('Focused diagnostic subject mismatch.')
if diagnostic.get('likely_failure_origin') != (
    'source_cow_geometry_or_content_defect'
):
    raise RuntimeError('Focused diagnostic did not confirm a source CoW defect.')
if diagnostic.get('source_cow_detached') is not True:
    raise RuntimeError('Source CoW was not confirmed detached.')
if diagnostic.get('preprocessed_cow_detached') is not True:
    raise RuntimeError('Preprocessed CoW was not confirmed detached.')
if diagnostic.get('crop_influence_detected') is not True:
    raise RuntimeError('CoW crop influence was not confirmed.')
if diagnostic.get('target_arrays_read') is not False:
    raise RuntimeError('Focused diagnostic reports target-array access.')

manifest_df = pd.read_csv(PARENT_MANIFEST_PATH)
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Parent manifest has {len(manifest_df)} rows; expected 148.'
    )
if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError('Parent manifest contains duplicate subjects.')
split_counts = manifest_df['dev_split'].value_counts().to_dict()
if split_counts != EXPECTED_SPLITS:
    raise RuntimeError(f'Unexpected split counts: {split_counts}')

subject_rows = manifest_df.loc[
    manifest_df['subject_id'].astype(str) == SUBJECT_ID
]
if len(subject_rows) != 1:
    raise RuntimeError(f'Expected one manifest row for {SUBJECT_ID}.')
subject_row = subject_rows.iloc[0]
if str(subject_row['dev_split']) != 'train':
    raise RuntimeError(f'{SUBJECT_ID} is not a training subject.')

PARENT_NPZ_PATH = Path(subject_row['vascufate_v2_npz_path'])
SOURCE_NCCT_PATH = Path(subject_row['path_raw_ncct'])
for path in [PARENT_NPZ_PATH, SOURCE_NCCT_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

for path in [CORRECTION_ROOT, SUBJECT_OUTPUT_DIR, CONFIG_PATH.parent]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

parent_npz_sha256 = file_sha256(PARENT_NPZ_PATH)
diagnostic_sha256 = file_sha256(DIAGNOSTIC_SUMMARY_PATH)
parent_manifest_sha256 = file_sha256(PARENT_MANIFEST_PATH)


with np.load(PARENT_NPZ_PATH, allow_pickle=False) as parent:
    required_keys = {
        'image',
        'perfusion_valid',
        'lvo_proximity',
        'input_channels',
        'affine',
        'crop_origin',
        'subject_id',
    }
    missing_keys = sorted(required_keys - set(parent.files))
    if missing_keys:
        raise RuntimeError(f'Parent NPZ missing keys: {missing_keys}')
    if str(parent['subject_id'].item()) != SUBJECT_ID:
        raise RuntimeError('Parent NPZ subject identifier mismatch.')

    parent_image_shape = tuple(parent['image'].shape)
    parent_spatial_shape = tuple(parent_image_shape[1:])
    input_channels = [str(value) for value in parent['input_channels'].tolist()]
    perfusion_valid = parent['perfusion_valid'].astype(bool)
    lvo_proximity = parent['lvo_proximity'].astype(np.float32)
    parent_affine = parent['affine'].astype(np.float64)
    parent_crop_origin = parent['crop_origin'].astype(np.int64)

if tuple(perfusion_valid.shape) != parent_spatial_shape:
    raise RuntimeError('Parent perfusion-valid shape mismatch.')
if tuple(lvo_proximity.shape) != parent_spatial_shape:
    raise RuntimeError('Parent LVO-proximity shape mismatch.')
if 'ncct_broad_window' not in input_channels:
    raise RuntimeError('Parent NPZ lacks ncct_broad_window.')

spacing = np.sqrt(np.sum(parent_affine[:3, :3] ** 2, axis=0))
if not np.allclose(spacing, TARGET_SPACING, atol=1e-4, rtol=0):
    raise RuntimeError(f'Parent spacing is {spacing.tolist()}.')

source_ncct = effective_canonical_image(SOURCE_NCCT_PATH)
ncct_on_parent_grid = resample_from_to(
    source_ncct,
    (parent_spatial_shape, parent_affine),
    order=1,
    mode='constant',
    cval=-1000.0,
)
ncct_hu = np.asarray(ncct_on_parent_grid.dataobj, dtype=np.float32)

corrected_crop_support_parent = construct_crop_support_without_cow(
    ncct_hu=ncct_hu,
    perfusion_valid=perfusion_valid,
    lvo_proximity=lvo_proximity,
)
crop_slices, local_start, local_stop = bounding_slices_with_margin(
    corrected_crop_support_parent,
    spacing=spacing,
    margin_mm=CROP_MARGIN_MM,
)
corrected_spatial_shape = tuple(
    int(stop - start) for start, stop in zip(local_start, local_stop)
)

if corrected_spatial_shape == parent_spatial_shape:
    raise RuntimeError('Correction did not reduce the CoW-influenced crop.')
if np.prod(corrected_spatial_shape) >= np.prod(parent_spatial_shape):
    raise RuntimeError('Corrected crop is not smaller than the parent crop.')

corrected_affine = translated_affine(parent_affine, local_start)
corrected_crop_origin = parent_crop_origin + local_start
corrected_crop_mask = corrected_crop_support_parent[crop_slices]


with np.load(PARENT_NPZ_PATH, allow_pickle=False) as parent:
    corrected = {
        key: crop_array(parent[key], parent_spatial_shape, crop_slices)
        for key in parent.files
    }

parent_target_binary_sum = int(np.asarray(corrected['target_binary']).sum())
parent_target_soft_sum_after_crop = float(
    np.asarray(corrected['target_soft'], dtype=np.float64).sum()
)

with np.load(PARENT_NPZ_PATH, allow_pickle=False) as parent:
    original_target_binary_sum = int(parent['target_binary'].sum())
    original_target_soft_sum = float(
        parent['target_soft'].astype(np.float64).sum()
    )

binary_target_preserved = (
    parent_target_binary_sum == original_target_binary_sum
)
soft_target_preserved = bool(
    np.isclose(
        parent_target_soft_sum_after_crop,
        original_target_soft_sum,
        atol=1e-3,
        rtol=0,
    )
)
if not binary_target_preserved or not soft_target_preserved:
    raise RuntimeError(
        'The target-independent corrected crop would remove target content. '
        'No corrected file was written.'
    )

corrected['crop_mask'] = corrected_crop_mask.astype(np.uint8)
corrected['cow_label'] = np.zeros(corrected_spatial_shape, dtype=np.uint8)
corrected['cow_soft_union'] = np.zeros(
    corrected_spatial_shape,
    dtype=np.float16,
)
corrected['vessel_features_raw'] = np.zeros_like(
    corrected['vessel_features_raw'],
    dtype=np.float32,
)
corrected['vessel_presence'] = np.zeros_like(
    corrected['vessel_presence'],
    dtype=np.uint8,
)
corrected['affine'] = corrected_affine.astype(np.float64)
corrected['crop_origin'] = corrected_crop_origin.astype(np.int32)
corrected['preprocessing_version'] = np.asarray(
    CORRECTED_PREPROCESSING_VERSION
)
corrected['cow_prior_valid'] = np.asarray(False, dtype=np.bool_)
corrected['cow_source_geometry_valid'] = np.asarray(False, dtype=np.bool_)
corrected['cow_prior_invalid_reason'] = np.asarray(
    'source mask detached 70-74 mm from brain; no defensible correction'
)
corrected['parent_preprocessing_version'] = np.asarray(
    PARENT_PREPROCESSING_VERSION
)
corrected['parent_npz_path'] = np.asarray(str(PARENT_NPZ_PATH))
corrected['parent_npz_sha256'] = np.asarray(parent_npz_sha256)
corrected['correction_version'] = np.asarray(CORRECTION_VERSION)
corrected['correction_local_start'] = local_start.astype(np.int32)
corrected['correction_local_stop'] = local_stop.astype(np.int32)
corrected['crop_was_target_independent'] = np.asarray(True, dtype=np.bool_)

corrected_image = np.asarray(corrected['image'])
if tuple(corrected_image.shape[1:]) != corrected_spatial_shape:
    raise RuntimeError('Corrected image shape does not match corrected crop.')

spatial_keys = [
    'target_binary',
    'target_soft',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
]
for key in spatial_keys:
    if tuple(np.asarray(corrected[key]).shape) != corrected_spatial_shape:
        raise RuntimeError(f'Corrected {key} shape mismatch.')

if not np.isfinite(corrected_image).all():
    raise RuntimeError('Corrected image contains nonfinite values.')
if not np.isfinite(corrected['target_soft']).all():
    raise RuntimeError('Corrected soft target contains nonfinite values.')
if not np.isfinite(corrected['lvo_proximity']).all():
    raise RuntimeError('Corrected LVO proximity contains nonfinite values.')
if np.count_nonzero(corrected['cow_label']) != 0:
    raise RuntimeError('Corrected CoW labels are not empty.')
if np.count_nonzero(corrected['cow_soft_union']) != 0:
    raise RuntimeError('Corrected CoW soft union is not empty.')
if np.count_nonzero(corrected['vessel_features_raw']) != 0:
    raise RuntimeError('Corrected vessel features are not empty.')
if np.count_nonzero(corrected['vessel_presence']) != 0:
    raise RuntimeError('Corrected vessel presence is not empty.')
if not bool(corrected['lvo_present'].item()):
    raise RuntimeError('Valid LVO state was not preserved.')

temporary_path = Path(str(CORRECTED_NPZ_PATH) + '.tmp')
with temporary_path.open('wb') as handle:
    np.savez_compressed(handle, **corrected)
os.replace(temporary_path, CORRECTED_NPZ_PATH)

corrected_npz_sha256 = file_sha256(CORRECTED_NPZ_PATH)


with np.load(CORRECTED_NPZ_PATH, allow_pickle=False) as check:
    committed_shape = tuple(check['image'].shape[1:])
    committed_version = str(check['preprocessing_version'].item())
    committed_cow_valid = bool(check['cow_prior_valid'].item())
    committed_binary_sum = int(check['target_binary'].sum())
    committed_soft_sum = float(check['target_soft'].astype(np.float64).sum())
    committed_lvo_present = bool(check['lvo_present'].item())

commit_checks = {
    'corrected_shape_matches': committed_shape == corrected_spatial_shape,
    'corrected_version_matches': (
        committed_version == CORRECTED_PREPROCESSING_VERSION
    ),
    'cow_prior_is_invalid': committed_cow_valid is False,
    'cow_arrays_are_empty': bool(
        np.count_nonzero(corrected['cow_label']) == 0
        and np.count_nonzero(corrected['cow_soft_union']) == 0
    ),
    'vessel_fields_are_empty': bool(
        np.count_nonzero(corrected['vessel_features_raw']) == 0
        and np.count_nonzero(corrected['vessel_presence']) == 0
    ),
    'lvo_state_preserved': committed_lvo_present,
    'binary_target_preserved_exactly': (
        committed_binary_sum == original_target_binary_sum
    ),
    'soft_target_preserved_within_1e_3': bool(
        np.isclose(committed_soft_sum, original_target_soft_sum, atol=1e-3, rtol=0)
    ),
    'crop_was_target_independent': True,
}
if not all(commit_checks.values()):
    failed = [key for key, value in commit_checks.items() if not value]
    raise RuntimeError(f'Corrected NPZ commit checks failed: {failed}')

candidate_manifest = manifest_df.copy()
candidate_manifest['trial8_dataset_revision'] = 'r2_cow_qc'
candidate_manifest['cow_prior_valid'] = True
candidate_manifest['cow_prior_action'] = 'use_intracranially_clipped_source_prior'
candidate_manifest['effective_preprocessing_version'] = (
    PARENT_PREPROCESSING_VERSION
)

subject_mask = candidate_manifest['subject_id'].astype(str) == SUBJECT_ID
candidate_manifest.loc[subject_mask, 'vascufate_v2_npz_path'] = str(
    CORRECTED_NPZ_PATH
)
candidate_manifest.loc[subject_mask, 'vascufate_v2_image_shape'] = str(
    tuple(corrected_image.shape)
)
candidate_manifest.loc[subject_mask, 'vascufate_v2_vessels_present'] = 0
candidate_manifest.loc[subject_mask, 'cow_prior_valid'] = False
candidate_manifest.loc[subject_mask, 'cow_prior_action'] = (
    'disabled_source_geometry_defect'
)
candidate_manifest.loc[subject_mask, 'effective_preprocessing_version'] = (
    CORRECTED_PREPROCESSING_VERSION
)

if len(candidate_manifest) != EXPECTED_SUBJECTS:
    raise RuntimeError('Candidate r2 manifest subject count changed.')
if candidate_manifest['subject_id'].duplicated().any():
    raise RuntimeError('Candidate r2 manifest contains duplicate subjects.')
if candidate_manifest['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
    raise RuntimeError('Candidate r2 manifest split counts changed.')
if int((~candidate_manifest['cow_prior_valid'].astype(bool)).sum()) != 1:
    raise RuntimeError('Candidate r2 manifest must disable exactly one CoW prior.')

candidate_manifest.to_csv(CANDIDATE_MANIFEST_PATH, index=False)

config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'corrected_preprocessing_version': CORRECTED_PREPROCESSING_VERSION,
    'parent_preprocessing_version': PARENT_PREPROCESSING_VERSION,
    'correction_version': CORRECTION_VERSION,
    'corrected_subject': SUBJECT_ID,
    'unchanged_parent_subjects': 147,
    'invalid_prior': 'CoW only',
    'lvo_policy': 'preserve valid source-derived LVO proximity',
    'crop_inputs': ['NCCT', 'perfusion_valid', 'lvo_proximity'],
    'crop_uses_cow': False,
    'crop_uses_target': False,
    'target_read_after_crop_freeze_for_preservation_check': True,
    'parent_manifest_path': str(PARENT_MANIFEST_PATH),
    'parent_manifest_sha256': parent_manifest_sha256,
    'diagnostic_summary_path': str(DIAGNOSTIC_SUMMARY_PATH),
    'diagnostic_summary_sha256': diagnostic_sha256,
    'parent_npz_path': str(PARENT_NPZ_PATH),
    'parent_npz_sha256': parent_npz_sha256,
    'corrected_npz_path': str(CORRECTED_NPZ_PATH),
    'corrected_npz_sha256': corrected_npz_sha256,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH),
    'bulk_atlas_registration_authorized': False,
}
atomic_write_json(config, CONFIG_PATH)

audit = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'subject_id': SUBJECT_ID,
    'parent_spatial_shape': list(parent_spatial_shape),
    'corrected_spatial_shape': list(corrected_spatial_shape),
    'local_crop_start': local_start.tolist(),
    'local_crop_stop': local_stop.tolist(),
    'parent_crop_origin': parent_crop_origin.tolist(),
    'corrected_crop_origin': corrected_crop_origin.tolist(),
    'parent_voxels': int(np.prod(parent_spatial_shape)),
    'corrected_voxels': int(np.prod(corrected_spatial_shape)),
    'voxel_reduction_fraction': float(
        1.0 - np.prod(corrected_spatial_shape) / np.prod(parent_spatial_shape)
    ),
    'original_target_binary_sum': original_target_binary_sum,
    'corrected_target_binary_sum': committed_binary_sum,
    'original_target_soft_sum': original_target_soft_sum,
    'corrected_target_soft_sum': committed_soft_sum,
    'cow_prior_valid': False,
    'lvo_present': committed_lvo_present,
    'crop_was_target_independent': True,
    'target_read_for_preservation_check_only': True,
    'commit_checks': commit_checks,
    'parent_npz_path': str(PARENT_NPZ_PATH),
    'parent_npz_sha256': parent_npz_sha256,
    'corrected_npz_path': str(CORRECTED_NPZ_PATH),
    'corrected_npz_sha256': corrected_npz_sha256,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH),
    'candidate_manifest_sha256': file_sha256(CANDIDATE_MANIFEST_PATH),
    'bulk_atlas_registration_authorized': False,
}
atomic_write_json(audit, CORRECTION_AUDIT_PATH)

print('Trial8 r2 correction summary:')
display(pd.DataFrame([{
    'subject_id': SUBJECT_ID,
    'parent_shape': parent_spatial_shape,
    'corrected_shape': corrected_spatial_shape,
    'local_crop_start': tuple(local_start.tolist()),
    'local_crop_stop': tuple(local_stop.tolist()),
    'voxel_reduction_fraction': audit['voxel_reduction_fraction'],
    'cow_prior_valid': False,
    'lvo_present': committed_lvo_present,
    'binary_target_preserved': binary_target_preserved,
    'soft_target_preserved': soft_target_preserved,
}]))

print('\nCommit checks:')
display(pd.DataFrame([
    {'check': key, 'passed': value}
    for key, value in commit_checks.items()
]))

print('\n' + '=' * 80)
print('TRIAL8 SUB-STROKE0016 R2 CORRECTION COMPLETE')
print('=' * 80)
print(f'Parent NPZ:                         {PARENT_NPZ_PATH}')
print(f'Corrected NPZ:                      {CORRECTED_NPZ_PATH}')
print(f'Candidate r2 manifest:              {CANDIDATE_MANIFEST_PATH}')
print(f'CoW prior valid:                    False')
print(f'LVO state preserved:                {committed_lvo_present}')
print(f'Crop selected using target:         False')
print(f'Target preservation checks passed:  True')
print('Study trainable parameters:         0 (preprocessing correction only)')
print('Bulk atlas registration authorized: False')
print(f'Correction audit:                   {CORRECTION_AUDIT_PATH}')
print(f'Configuration:                      {CONFIG_PATH}')
print('\nNext: regenerate and verify the SynthStrip mask for the corrected crop.')


In [ ]:
import os
import sys
import json
import time
import shutil
import hashlib
import tempfile
import subprocess
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import ndimage
from IPython.display import display, Image as DisplayImage
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PARENT_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
CORRECTED_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
VERIFICATION_VERSION = 'v1_sub0016_r2_corrected_crop'
SUBJECT_ID = 'sub-stroke0016'

CANDIDATE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_candidate.csv'
)
PARENT_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
CORRECTION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / CORRECTED_PREPROCESSING_VERSION
)
CORRECTION_AUDIT_PATH = CORRECTION_ROOT / 'sub0016_correction_audit.json'

SOURCE_DIR = (
    PROJECT_ROOT
    / 'preprocessing'
    / PARENT_PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / 'source'
)
PROVENANCE_PATH = SOURCE_DIR / 'synthstrip_source_provenance.json'
SCRIPT_PATH = SOURCE_DIR / 'mri_synthstrip'
MODEL_PATH = SOURCE_DIR / 'synthstrip.1.pt'
REQUIREMENTS_PATH = SOURCE_DIR / 'gpu.requirements.txt'

VENV_ROOT = Path('/tmp/acf_net_synthstrip_py310')
VENV_PYTHON = VENV_ROOT / 'bin' / 'python'
VENV_READY_PATH = VENV_ROOT / 'trial8_synthstrip_environment.json'

UV_VERSION = '0.8.17'
PYTHON_VERSION = '3.10.18'
PYTORCH_INDEX = 'https://download.pytorch.org/whl/cu118'

VERIFICATION_ROOT = (
    CORRECTION_ROOT
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / VERIFICATION_VERSION
)
CASE_ROOT = VERIFICATION_ROOT / 'cases' / SUBJECT_ID
MASK_PATH = CASE_ROOT / 'ncct_synthstrip_mask.nii.gz'
QA_PATH = CASE_ROOT / 'corrected_crop_visual_qa.png'
AUDIT_CSV_PATH = VERIFICATION_ROOT / 'sub0016_r2_verification.csv'
GATE_PATH = VERIFICATION_ROOT / 'sub0016_r2_candidate_gate.json'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)
BORDER_MM = 1
MIN_BRAIN_VOLUME_ML = 500.0
MAX_BRAIN_VOLUME_ML = 2200.0
MIN_LARGEST_COMPONENT_FRACTION = 0.995
MIN_CROP_CONTAINMENT = 0.99


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def atomic_write_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)


def atomic_save_mask(mask, affine, path):
    temporary = path.with_name(path.name + '.tmp.nii.gz')
    image = nib.Nifti1Image(
        np.asarray(mask, dtype=np.uint8),
        np.asarray(affine, dtype=np.float64),
    )
    image.set_sform(np.asarray(affine, dtype=np.float64), code=2)
    image.set_qform(np.asarray(affine, dtype=np.float64), code=1)
    nib.save(image, str(temporary))
    temporary.replace(path)


def ensure_uv():
    try:
        installed = metadata.version('uv')
    except metadata.PackageNotFoundError:
        installed = None

    if installed != UV_VERSION:
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '--quiet',
            '--upgrade',
            f'uv=={UV_VERSION}',
        ])

    uv_path = shutil.which('uv')
    if uv_path is None:
        raise RuntimeError('uv installed but its executable was not found.')
    return Path(uv_path)


def synthstrip_gpu_smoke_test():
    smoke_code = r'''
import json
import torch
import numpy
import scipy
import nibabel
import surfa

record = {
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "nibabel": nibabel.__version__,
}
print(json.dumps(record))
if not torch.cuda.is_available():
    raise SystemExit("CUDA is unavailable inside the SynthStrip environment.")
'''
    result = subprocess.run(
        [str(VENV_PYTHON), '-c', smoke_code],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        return None, result
    lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not lines:
        return None, result
    return json.loads(lines[-1]), result


def ensure_synthstrip_environment():
    reused = bool(VENV_PYTHON.exists() and VENV_READY_PATH.exists())
    gpu_record = None

    if reused:
        gpu_record, _ = synthstrip_gpu_smoke_test()
        if gpu_record is None:
            reused = False

    if not reused:
        if VENV_ROOT.exists():
            shutil.rmtree(VENV_ROOT)

        uv_path = ensure_uv()
        subprocess.check_call([
            str(uv_path),
            'venv',
            '--python',
            PYTHON_VERSION,
            '--seed',
            str(VENV_ROOT),
        ])
        subprocess.check_call([
            str(VENV_PYTHON),
            '-m',
            'pip',
            'install',
            '--quiet',
            '--no-cache-dir',
            '--extra-index-url',
            PYTORCH_INDEX,
            '-r',
            str(REQUIREMENTS_PATH),
        ])

        gpu_record, result = synthstrip_gpu_smoke_test()
        if gpu_record is None:
            raise RuntimeError(
                'SynthStrip GPU environment smoke test failed:\n'
                + result.stdout
                + '\n'
                + result.stderr
            )

        environment_record = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'venv_root': str(VENV_ROOT),
            'python_version': PYTHON_VERSION,
            'uv_version': UV_VERSION,
            'source_provenance_path': str(PROVENANCE_PATH),
            'source_provenance_sha256': file_sha256(PROVENANCE_PATH),
            'script_path': str(SCRIPT_PATH),
            'model_path': str(MODEL_PATH),
            'gpu_smoke_test': gpu_record,
            'created_by': VERIFICATION_VERSION,
        }
        VENV_READY_PATH.write_text(
            json.dumps(environment_record, indent=2),
            encoding='utf-8',
        )

    return reused, gpu_record


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == 'true'


def translated_affine(affine, voxel_offset):
    transform = np.eye(4, dtype=np.float64)
    transform[:3, 3] = np.asarray(voxel_offset, dtype=np.float64)
    return np.asarray(affine, dtype=np.float64) @ transform


def exact_slice(parent_array, corrected_array, spatial_shape, crop_slices):
    parent_array = np.asarray(parent_array)
    corrected_array = np.asarray(corrected_array)
    if tuple(parent_array.shape) == tuple(spatial_shape):
        expected = parent_array[crop_slices]
    elif (
        parent_array.ndim == 4
        and tuple(parent_array.shape[1:]) == tuple(spatial_shape)
    ):
        expected = parent_array[(slice(None), *crop_slices)]
    else:
        expected = parent_array
    return bool(np.array_equal(expected, corrected_array))


def normalized_slice(array):
    values = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values, dtype=np.float32)
    low, high = np.percentile(values[finite], [1, 99])
    if high <= low:
        return np.zeros_like(values, dtype=np.float32)
    return np.clip((values - low) / (high - low), 0.0, 1.0)


def contour_if_present(axis, mask, color, linewidth=1.0):
    mask = np.asarray(mask, dtype=bool)
    if mask.any() and (~mask).any():
        axis.contour(mask.astype(np.uint8), levels=[0.5], colors=[color],
                     linewidths=linewidth)


def select_qa_indices(brain_mask, lvo_mask):
    brain_coordinates = np.argwhere(brain_mask)
    if not len(brain_coordinates):
        raise RuntimeError('Cannot select QA slices from an empty brain mask.')
    center = np.rint(np.median(brain_coordinates, axis=0)).astype(int)
    if lvo_mask.any():
        lvo_center = np.rint(
            np.median(np.argwhere(lvo_mask), axis=0)
        ).astype(int)
        center = np.rint((center + lvo_center) / 2.0).astype(int)
    return tuple(
        int(np.clip(center[axis], 0, brain_mask.shape[axis] - 1))
        for axis in range(3)
    )


def save_visual_qa(broad_window, brain_mask, lvo_mask, output_path):
    x_index, y_index, z_index = select_qa_indices(brain_mask, lvo_mask)
    planes = [
        (
            'sagittal',
            np.rot90(broad_window[x_index, :, :]),
            np.rot90(brain_mask[x_index, :, :]),
            np.rot90(lvo_mask[x_index, :, :]),
            x_index,
        ),
        (
            'coronal',
            np.rot90(broad_window[:, y_index, :]),
            np.rot90(brain_mask[:, y_index, :]),
            np.rot90(lvo_mask[:, y_index, :]),
            y_index,
        ),
        (
            'axial',
            np.rot90(broad_window[:, :, z_index]),
            np.rot90(brain_mask[:, :, z_index]),
            np.rot90(lvo_mask[:, :, z_index]),
            z_index,
        ),
    ]

    figure, axes = plt.subplots(2, 3, figsize=(15, 9))
    for column, (name, image, brain, lvo, index) in enumerate(planes):
        display_image = normalized_slice(image)
        axes[0, column].imshow(display_image, cmap='gray', origin='lower')
        contour_if_present(axes[0, column], brain, 'red', 1.2)
        axes[0, column].set_title(
            f'{name} {index}: corrected SynthStrip boundary'
        )

        axes[1, column].imshow(display_image, cmap='gray', origin='lower')
        contour_if_present(axes[1, column], brain, 'red', 1.0)
        contour_if_present(axes[1, column], lvo, 'cyan', 1.2)
        axes[1, column].set_title(
            f'{name} {index}: retained LVO, CoW disabled'
        )

        axes[0, column].axis('off')
        axes[1, column].axis('off')

    figure.suptitle(
        f'{SUBJECT_ID}: r2 corrected crop and brain mask',
        fontsize=15,
    )
    figure.tight_layout(rect=[0, 0, 1, 0.96])
    figure.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(figure)


required_paths = [
    CANDIDATE_MANIFEST_PATH,
    PARENT_MANIFEST_PATH,
    CORRECTION_AUDIT_PATH,
    PROVENANCE_PATH,
    SCRIPT_PATH,
    MODEL_PATH,
    REQUIREMENTS_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 r2 SynthStrip verification prerequisites are missing. '
        'Run the r2 correction cell first. The isolated runtime environment '
        'will be rebuilt automatically, but the frozen Drive assets must '
        'exist. Missing:\n'
        + '\n'.join(missing_paths)
    )

candidate_manifest = pd.read_csv(CANDIDATE_MANIFEST_PATH)
parent_manifest = pd.read_csv(PARENT_MANIFEST_PATH)
for name, frame in [
    ('candidate', candidate_manifest),
    ('parent', parent_manifest),
]:
    if len(frame) != EXPECTED_SUBJECTS:
        raise RuntimeError(f'{name} manifest has {len(frame)} rows.')
    if frame['subject_id'].duplicated().any():
        raise RuntimeError(f'{name} manifest has duplicate subjects.')
    if frame['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
        raise RuntimeError(f'{name} manifest split counts changed.')

candidate_rows = candidate_manifest.loc[
    candidate_manifest['subject_id'].astype(str) == SUBJECT_ID
]
parent_rows = parent_manifest.loc[
    parent_manifest['subject_id'].astype(str) == SUBJECT_ID
]
if len(candidate_rows) != 1 or len(parent_rows) != 1:
    raise RuntimeError(f'Expected one row for {SUBJECT_ID} in each manifest.')

candidate_row = candidate_rows.iloc[0]
parent_row = parent_rows.iloc[0]
if str(candidate_row['dev_split']) != 'train':
    raise RuntimeError(f'{SUBJECT_ID} is not a training subject.')
if as_bool(candidate_row['cow_prior_valid']):
    raise RuntimeError('Candidate manifest did not disable the CoW prior.')

CORRECTED_NPZ_PATH = Path(candidate_row['vascufate_v2_npz_path'])
PARENT_NPZ_PATH = Path(parent_row['vascufate_v2_npz_path'])
for path in [CORRECTED_NPZ_PATH, PARENT_NPZ_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

correction_audit = json.loads(
    CORRECTION_AUDIT_PATH.read_text(encoding='utf-8')
)
if correction_audit.get('subject_id') != SUBJECT_ID:
    raise RuntimeError('Correction audit subject mismatch.')
if not all(correction_audit.get('commit_checks', {}).values()):
    raise RuntimeError('The r2 correction commit checks did not all pass.')
if correction_audit.get('bulk_atlas_registration_authorized') is not False:
    raise RuntimeError('Atlas registration must remain locked.')
if correction_audit.get('corrected_npz_sha256') != file_sha256(
    CORRECTED_NPZ_PATH
):
    raise RuntimeError('Corrected NPZ no longer matches its correction audit.')

provenance = json.loads(PROVENANCE_PATH.read_text(encoding='utf-8'))
if provenance.get('script_sha256') != file_sha256(SCRIPT_PATH):
    raise RuntimeError('SynthStrip script does not match frozen provenance.')
if provenance.get('model_sha256') != file_sha256(MODEL_PATH):
    raise RuntimeError('SynthStrip model does not match frozen provenance.')
if provenance.get('requirements_sha256') != file_sha256(REQUIREMENTS_PATH):
    raise RuntimeError(
        'SynthStrip requirements do not match frozen provenance.'
    )

environment_reused, environment_gpu_record = ensure_synthstrip_environment()

gpu_result = subprocess.run(
    [
        str(VENV_PYTHON),
        '-c',
        'import torch; '
        'print(torch.cuda.get_device_name(0) if torch.cuda.is_available() '
        'else "CUDA unavailable"); '
        'raise SystemExit(0 if torch.cuda.is_available() else 1)',
    ],
    text=True,
    capture_output=True,
)
if gpu_result.returncode != 0:
    raise RuntimeError(
        'CUDA is unavailable in the isolated SynthStrip environment.\n'
        + gpu_result.stdout
        + '\n'
        + gpu_result.stderr
    )

with np.load(PARENT_NPZ_PATH, allow_pickle=False) as parent:
    parent_spatial_shape = tuple(parent['image'].shape[1:])
    parent_affine = parent['affine'].astype(np.float64)
    parent_keys = set(parent.files)

with np.load(CORRECTED_NPZ_PATH, allow_pickle=False) as corrected:
    required_keys = {
        'image',
        'input_channels',
        'crop_mask',
        'target_binary',
        'target_soft',
        'perfusion_valid',
        'mtt_valid',
        'mtt_interpolation_support',
        'cow_label',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
        'vessel_features_raw',
        'vessel_presence',
        'affine',
        'cow_prior_valid',
        'correction_local_start',
        'correction_local_stop',
        'crop_was_target_independent',
        'subject_id',
    }
    missing_keys = sorted(required_keys - set(corrected.files))
    if missing_keys:
        raise RuntimeError(f'Corrected NPZ missing keys: {missing_keys}')
    if str(corrected['subject_id'].item()) != SUBJECT_ID:
        raise RuntimeError('Corrected NPZ subject mismatch.')

    corrected_keys = set(corrected.files)
    corrected_image = corrected['image'].astype(np.float32)
    channel_names = [str(value) for value in corrected['input_channels']]
    broad_index = channel_names.index('ncct_broad_window')
    broad_window = corrected_image[broad_index]
    crop_mask = corrected['crop_mask'].astype(bool)
    lvo_mask = corrected['lvo_proximity'].astype(np.float32) >= 0.5
    corrected_affine = corrected['affine'].astype(np.float64)
    local_start = corrected['correction_local_start'].astype(np.int64)
    local_stop = corrected['correction_local_stop'].astype(np.int64)
    cow_prior_valid = bool(corrected['cow_prior_valid'].item())
    crop_was_target_independent = bool(
        corrected['crop_was_target_independent'].item()
    )
    lvo_present = bool(corrected['lvo_present'].item())
    cow_voxels = int(np.count_nonzero(corrected['cow_label']))
    cow_soft_voxels = int(np.count_nonzero(corrected['cow_soft_union']))
    vessel_feature_nonzero = int(
        np.count_nonzero(corrected['vessel_features_raw'])
    )
    vessel_presence_count = int(
        np.count_nonzero(corrected['vessel_presence'])
    )

corrected_spatial_shape = tuple(corrected_image.shape[1:])
crop_slices = tuple(
    slice(int(start), int(stop))
    for start, stop in zip(local_start, local_stop)
)
if tuple(
    int(stop - start) for start, stop in zip(local_start, local_stop)
) != corrected_spatial_shape:
    raise RuntimeError('Correction bounds do not match corrected shape.')

retained_spatial_keys = [
    'image',
    'target_binary',
    'target_soft',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'lvo_proximity',
]
exact_slice_checks = {}
with np.load(PARENT_NPZ_PATH, allow_pickle=False) as parent, np.load(
    CORRECTED_NPZ_PATH,
    allow_pickle=False,
) as corrected:
    for key in retained_spatial_keys:
        if key not in parent_keys or key not in corrected_keys:
            raise RuntimeError(f'Cannot compare retained key: {key}')
        exact_slice_checks[f'{key}_is_exact_parent_slice'] = exact_slice(
            parent[key],
            corrected[key],
            parent_spatial_shape,
            crop_slices,
        )

expected_affine = translated_affine(parent_affine, local_start)
spacing = np.sqrt(np.sum(corrected_affine[:3, :3] ** 2, axis=0))
orientation = ''.join(nib.aff2axcodes(corrected_affine))

pre_synthstrip_checks = {
    **exact_slice_checks,
    'corrected_key_set_contains_parent_key_set': bool(
        parent_keys.issubset(corrected_keys)
    ),
    'corrected_affine_is_parent_affine_plus_crop_offset': bool(
        np.allclose(corrected_affine, expected_affine, atol=1e-7, rtol=0)
    ),
    'orientation_is_ras': orientation == 'RAS',
    'spacing_is_0p8_0p8_2p0': bool(
        np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0)
    ),
    'cow_prior_is_disabled': cow_prior_valid is False,
    'cow_label_is_empty': cow_voxels == 0,
    'cow_soft_union_is_empty': cow_soft_voxels == 0,
    'vessel_features_are_empty': vessel_feature_nonzero == 0,
    'vessel_presence_is_empty': vessel_presence_count == 0,
    'lvo_state_is_preserved': lvo_present,
    'crop_was_target_independent': crop_was_target_independent,
}
if not all(pre_synthstrip_checks.values()):
    failed = [key for key, value in pre_synthstrip_checks.items() if not value]
    raise RuntimeError(f'Pre-SynthStrip r2 checks failed: {failed}')

for path in [VERIFICATION_ROOT, CASE_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

print('Trial8 r2 corrected-case SynthStrip verification')
print(
    'SynthStrip environment: '
    + ('reused' if environment_reused else 'rebuilt automatically')
)
print(f'GPU: {gpu_result.stdout.strip()}')
print(f'Corrected NPZ: {CORRECTED_NPZ_PATH}')
print('CoW prior valid: False')
print('Target used to select crop: False')
print('Study trainable parameters: 0 (preprocessing verification only)')
print('Bulk atlas registration authorized: False')

run_start = time.time()
with tempfile.TemporaryDirectory(
    prefix='trial8_r2_sub0016_synthstrip_',
    dir='/tmp',
) as temporary_directory:
    temporary_directory = Path(temporary_directory)
    input_path = temporary_directory / 'ncct_broad_window.nii.gz'
    brain_path = temporary_directory / 'ncct_brain.nii.gz'
    temporary_mask_path = temporary_directory / 'ncct_mask.nii.gz'

    input_image = nib.Nifti1Image(broad_window, corrected_affine)
    input_image.set_sform(corrected_affine, code=2)
    input_image.set_qform(corrected_affine, code=1)
    nib.save(input_image, str(input_path))

    command = [
        str(VENV_PYTHON),
        str(SCRIPT_PATH),
        '-i',
        str(input_path),
        '-o',
        str(brain_path),
        '-m',
        str(temporary_mask_path),
        '-b',
        str(BORDER_MM),
        '-g',
        '--model',
        str(MODEL_PATH),
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            'SynthStrip failed.\nSTDOUT:\n'
            + result.stdout[-4000:]
            + '\nSTDERR:\n'
            + result.stderr[-4000:]
        )

    temporary_mask_image = nib.load(str(temporary_mask_path))
    temporary_mask_array = np.asanyarray(temporary_mask_image.dataobj)
    if temporary_mask_array.ndim == 4 and temporary_mask_array.shape[-1] == 1:
        temporary_mask_array = temporary_mask_array[..., 0]
    brain_mask = temporary_mask_array > 0.5

    if brain_mask.shape != corrected_spatial_shape:
        raise RuntimeError(
            f'Mask shape {brain_mask.shape} does not match '
            f'{corrected_spatial_shape}.'
        )
    if not np.allclose(
        temporary_mask_image.affine,
        corrected_affine,
        atol=1e-5,
        rtol=1e-5,
    ):
        raise RuntimeError('SynthStrip mask affine does not match r2 grid.')

    atomic_save_mask(brain_mask, corrected_affine, MASK_PATH)

elapsed_seconds = float(time.time() - run_start)
saved_mask_image = nib.load(str(MASK_PATH))
saved_mask_array = np.asanyarray(saved_mask_image.dataobj)
if saved_mask_array.ndim == 4 and saved_mask_array.shape[-1] == 1:
    saved_mask_array = saved_mask_array[..., 0]
saved_brain_mask = saved_mask_array > 0.5
saved_mask_shape_matches = saved_brain_mask.shape == corrected_spatial_shape
saved_mask_affine_matches = bool(
    np.allclose(
        saved_mask_image.affine,
        corrected_affine,
        atol=1e-5,
        rtol=1e-5,
    )
)
saved_mask_values_match = bool(np.array_equal(saved_brain_mask, brain_mask))

labeled, component_count = ndimage.label(brain_mask)
component_sizes = np.bincount(labeled.ravel())[1:]
largest_component_fraction = (
    float(component_sizes.max() / brain_mask.sum())
    if len(component_sizes) and brain_mask.any()
    else 0.0
)
voxel_volume_ml = float(np.prod(spacing) / 1000.0)
brain_voxels = int(brain_mask.sum())
brain_volume_ml = float(brain_voxels * voxel_volume_ml)
crop_containment = float(
    np.logical_and(brain_mask, crop_mask).sum() / max(brain_voxels, 1)
)
lvo_voxels = int(lvo_mask.sum())
lvo_coverage = (
    float(np.logical_and(lvo_mask, brain_mask).sum() / lvo_voxels)
    if lvo_voxels
    else 1.0
)
edge_touch_x = bool(brain_mask[0].any() or brain_mask[-1].any())
edge_touch_y = bool(brain_mask[:, 0].any() or brain_mask[:, -1].any())
edge_touch_z = bool(brain_mask[:, :, 0].any() or brain_mask[:, :, -1].any())

mask_checks = {
    'mask_nonempty': bool(brain_mask.any()),
    'brain_volume_between_500_and_2200_ml': bool(
        MIN_BRAIN_VOLUME_ML <= brain_volume_ml <= MAX_BRAIN_VOLUME_ML
    ),
    'largest_component_fraction_at_least_0p995': bool(
        largest_component_fraction >= MIN_LARGEST_COMPONENT_FRACTION
    ),
    'brain_mask_at_least_0p99_inside_crop_mask': bool(
        crop_containment >= MIN_CROP_CONTAINMENT
    ),
    'does_not_touch_x_or_y_crop_edges': bool(
        not (edge_touch_x or edge_touch_y)
    ),
    'saved_mask_shape_matches_r2': saved_mask_shape_matches,
    'saved_mask_affine_matches_r2': saved_mask_affine_matches,
    'saved_mask_values_match_generated_mask': saved_mask_values_match,
}

automated_safety_passed = bool(
    all(pre_synthstrip_checks.values()) and all(mask_checks.values())
)

save_visual_qa(
    broad_window=broad_window,
    brain_mask=brain_mask,
    lvo_mask=lvo_mask,
    output_path=QA_PATH,
)

audit_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'verification_version': VERIFICATION_VERSION,
    'subject_id': SUBJECT_ID,
    'dev_split': str(candidate_row['dev_split']),
    'corrected_npz_path': str(CORRECTED_NPZ_PATH),
    'corrected_npz_sha256': file_sha256(CORRECTED_NPZ_PATH),
    'mask_path': str(MASK_PATH),
    'mask_sha256': file_sha256(MASK_PATH),
    'qa_path': str(QA_PATH),
    'orientation': orientation,
    'spacing_x_mm': float(spacing[0]),
    'spacing_y_mm': float(spacing[1]),
    'spacing_z_mm': float(spacing[2]),
    'shape_x': int(brain_mask.shape[0]),
    'shape_y': int(brain_mask.shape[1]),
    'shape_z': int(brain_mask.shape[2]),
    'brain_voxels': brain_voxels,
    'brain_volume_ml': brain_volume_ml,
    'component_count': int(component_count),
    'largest_component_fraction': largest_component_fraction,
    'crop_containment_fraction': crop_containment,
    'lvo_proximity_voxels': lvo_voxels,
    'lvo_coverage_fraction_informational': lvo_coverage,
    'cow_voxels': cow_voxels,
    'cow_prior_valid': False,
    'edge_touch_x': edge_touch_x,
    'edge_touch_y': edge_touch_y,
    'edge_touch_z': edge_touch_z,
    'elapsed_seconds': elapsed_seconds,
    'compute': 'GPU',
    'target_used_to_select_crop': False,
    'target_arrays_read_for_preservation_check_only': True,
    'automated_safety_passed': automated_safety_passed,
    'manual_visual_qa_approved': False,
    'r2_frozen': False,
    'bulk_atlas_registration_authorized': False,
}
pd.DataFrame([audit_record]).to_csv(AUDIT_CSV_PATH, index=False)

gate = {
    **audit_record,
    'pre_synthstrip_checks': pre_synthstrip_checks,
    'mask_checks': mask_checks,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH),
    'candidate_manifest_sha256': file_sha256(CANDIDATE_MANIFEST_PATH),
    'parent_manifest_path': str(PARENT_MANIFEST_PATH),
    'parent_manifest_sha256': file_sha256(PARENT_MANIFEST_PATH),
    'correction_audit_path': str(CORRECTION_AUDIT_PATH),
    'correction_audit_sha256': file_sha256(CORRECTION_AUDIT_PATH),
    'synthstrip_script_path': str(SCRIPT_PATH),
    'synthstrip_script_sha256': file_sha256(SCRIPT_PATH),
    'synthstrip_model_path': str(MODEL_PATH),
    'synthstrip_model_sha256': file_sha256(MODEL_PATH),
    'next_required_action': (
        'manual visual check of corrected r2 SynthStrip mask before freezing r2'
    ),
}
atomic_write_json(gate, GATE_PATH)

print('\nPre-SynthStrip integrity checks:')
display(pd.DataFrame([
    {'check': key, 'passed': value}
    for key, value in pre_synthstrip_checks.items()
]))

print('\nCorrected SynthStrip mask checks:')
display(pd.DataFrame([
    {'check': key, 'passed': value}
    for key, value in mask_checks.items()
]))

print('\nVerification metrics:')
display(pd.DataFrame([audit_record]))
display(DisplayImage(filename=str(QA_PATH)))

print('\n' + '=' * 80)
print('TRIAL8 SUB-STROKE0016 R2 SYNTHSTRIP VERIFICATION COMPLETE')
print('=' * 80)
print(f'Automated safety passed:              {automated_safety_passed}')
print('CoW prior valid:                      False')
print(f'LVO state preserved:                  {lvo_present}')
print(f'LVO coverage, informational:          {lvo_coverage:.6f}')
print('Target used to select crop:           False')
print('Target read for preservation check:   True')
print('Manual visual QA approved:            False')
print('R2 dataset frozen:                    False')
print('Bulk atlas registration authorized:   False')
print('Study trainable parameters:           0 (preprocessing verification only)')
print(f'Corrected mask:                       {MASK_PATH}')
print(f'Visual QA:                            {QA_PATH}')
print(f'Candidate gate:                       {GATE_PATH}')

if not automated_safety_passed:
    failed = [
        key
        for checks in [pre_synthstrip_checks, mask_checks]
        for key, value in checks.items()
        if not value
    ]
    raise RuntimeError(
        'TRIAL8 R2 SYNTHSTRIP VERIFICATION FAILED. Do not freeze r2 or begin '
        f'atlas registration. Failed checks: {failed}'
    )

print('\nStop here and check the corrected visual QA image before freezing r2.')


In [ ]:
import os
import json
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from scipy import ndimage
from IPython.display import display
from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PARENT_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
CORRECTED_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
R1_BULK_VERSION = 'bulk_all148_v1'
R2_VERIFICATION_VERSION = 'v1_sub0016_r2_corrected_crop'
FREEZE_VERSION = 'v1_r2_mixed_147r1_1r2'
CORRECTED_SUBJECT_ID = 'sub-stroke0016'

PARENT_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r1_frozen.csv'
)
CANDIDATE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_candidate.csv'
)
FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_frozen.csv'
)

R1_BULK_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PARENT_PREPROCESSING_VERSION
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / R1_BULK_VERSION
)
R1_CASE_ROOT = R1_BULK_ROOT / 'cases'
R1_VERIFICATION_ROOT = (
    R1_BULK_ROOT
    / 'verification'
    / 'v1_structural_mask_and_cow_compatibility'
)
R1_VERIFICATION_SUMMARY_PATH = (
    R1_VERIFICATION_ROOT / 'verification_summary.json'
)
R1_COW_AUDIT_PATH = R1_VERIFICATION_ROOT / 'cow_compatibility_audit.csv'

R2_ROOT = (
    PROJECT_ROOT / 'preprocessing' / CORRECTED_PREPROCESSING_VERSION
)
R2_CORRECTION_AUDIT_PATH = R2_ROOT / 'sub0016_correction_audit.json'
R2_VERIFICATION_ROOT = (
    R2_ROOT
    / 'brain_masks'
    / SYNTHSTRIP_VERSION
    / R2_VERIFICATION_VERSION
)
R2_MASK_PATH = (
    R2_VERIFICATION_ROOT
    / 'cases'
    / CORRECTED_SUBJECT_ID
    / 'ncct_synthstrip_mask.nii.gz'
)
R2_CANDIDATE_GATE_PATH = (
    R2_VERIFICATION_ROOT / 'sub0016_r2_candidate_gate.json'
)

FINAL_VERIFICATION_ROOT = R2_ROOT / 'final_verification' / FREEZE_VERSION
CASE_AUDIT_PATH = FINAL_VERIFICATION_ROOT / 'r2_case_integrity_audit.csv'
ERROR_PATH = FINAL_VERIFICATION_ROOT / 'r2_case_integrity_errors.csv'
FREEZE_GATE_PATH = FINAL_VERIFICATION_ROOT / 'r2_dataset_freeze_gate.json'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)
EXPECTED_IMAGE_CHANNELS = 15
MIN_BRAIN_VOLUME_ML = 500.0
MAX_BRAIN_VOLUME_ML = 2200.0
MIN_LARGEST_COMPONENT_FRACTION = 0.995
MIN_CROP_CONTAINMENT = 0.99


MANUAL_VISUAL_QA_APPROVED = True
MANUAL_VISUAL_QA_NOTE = (
    'Bulk r1 masks checked; sub-stroke0016 corrected r2 axial slice 34 '
    'checked and approved with the defective CoW prior disabled.'
)


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def atomic_write_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)


def atomic_write_csv(frame, path):
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == 'true'


def effective_3d_mask(path):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 3:
        raise RuntimeError(f'Mask has unsupported shape {array.shape}.')
    return image, array


required_paths = [
    PARENT_MANIFEST_PATH,
    CANDIDATE_MANIFEST_PATH,
    R1_VERIFICATION_SUMMARY_PATH,
    R1_COW_AUDIT_PATH,
    R2_CORRECTION_AUDIT_PATH,
    R2_CANDIDATE_GATE_PATH,
    R2_MASK_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 r2 freeze prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

for path in [FINAL_VERIFICATION_ROOT, FROZEN_MANIFEST_PATH.parent]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

parent_manifest = pd.read_csv(PARENT_MANIFEST_PATH)
candidate_manifest = pd.read_csv(CANDIDATE_MANIFEST_PATH)

required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'vascufate_v2_image_shape',
    'cow_prior_valid',
    'cow_prior_action',
    'effective_preprocessing_version',
}
missing_columns = sorted(
    required_manifest_columns - set(candidate_manifest.columns)
)
if missing_columns:
    raise RuntimeError(
        f'Candidate r2 manifest is missing columns: {missing_columns}'
    )

for name, frame in [
    ('parent', parent_manifest),
    ('candidate', candidate_manifest),
]:
    if len(frame) != EXPECTED_SUBJECTS:
        raise RuntimeError(
            f'{name} manifest has {len(frame)} rows; expected 148.'
        )
    if frame['subject_id'].duplicated().any():
        raise RuntimeError(f'{name} manifest contains duplicate subjects.')
    if frame['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
        raise RuntimeError(f'{name} manifest split counts changed.')

parent_lookup = parent_manifest.set_index('subject_id')
candidate_lookup = candidate_manifest.set_index('subject_id')
if set(parent_lookup.index) != set(candidate_lookup.index):
    raise RuntimeError('Parent and candidate subject sets differ.')

changed_subjects = []
for subject_id in sorted(candidate_lookup.index):
    parent_path = str(parent_lookup.loc[subject_id, 'vascufate_v2_npz_path'])
    candidate_path = str(
        candidate_lookup.loc[subject_id, 'vascufate_v2_npz_path']
    )
    if parent_path != candidate_path:
        changed_subjects.append(subject_id)
if changed_subjects != [CORRECTED_SUBJECT_ID]:
    raise RuntimeError(
        'Exactly sub-stroke0016 must differ from r1. '
        f'Observed: {changed_subjects}'
    )

disabled_cow_subjects = sorted(
    candidate_manifest.loc[
        ~candidate_manifest['cow_prior_valid'].map(as_bool),
        'subject_id',
    ].astype(str).tolist()
)
if disabled_cow_subjects != [CORRECTED_SUBJECT_ID]:
    raise RuntimeError(
        'Exactly sub-stroke0016 must have its CoW prior disabled. '
        f'Observed: {disabled_cow_subjects}'
    )

r1_summary = json.loads(
    R1_VERIFICATION_SUMMARY_PATH.read_text(encoding='utf-8')
)
if r1_summary.get('subjects_verified') != EXPECTED_SUBJECTS:
    raise RuntimeError('The r1 mask verification did not cover 148 subjects.')
if r1_summary.get('all_structural_masks_passed') is not True:
    raise RuntimeError('The r1 structural-mask verification did not pass.')
if r1_summary.get('target_arrays_read') is not False:
    raise RuntimeError('The r1 mask verification reports target access.')

r2_correction = json.loads(
    R2_CORRECTION_AUDIT_PATH.read_text(encoding='utf-8')
)
if r2_correction.get('subject_id') != CORRECTED_SUBJECT_ID:
    raise RuntimeError('The r2 correction audit subject does not match.')
if not all(r2_correction.get('commit_checks', {}).values()):
    raise RuntimeError('The r2 correction commit checks did not all pass.')
if r2_correction.get('cow_prior_valid') is not False:
    raise RuntimeError('The r2 correction did not disable the CoW prior.')
if r2_correction.get('crop_was_target_independent') is not True:
    raise RuntimeError('The r2 correction crop was not target-independent.')

r2_gate = json.loads(R2_CANDIDATE_GATE_PATH.read_text(encoding='utf-8'))
if r2_gate.get('subject_id') != CORRECTED_SUBJECT_ID:
    raise RuntimeError('The r2 candidate gate subject does not match.')
if r2_gate.get('automated_safety_passed') is not True:
    raise RuntimeError('The r2 corrected-mask automated safety did not pass.')
if not all(r2_gate.get('pre_synthstrip_checks', {}).values()):
    raise RuntimeError('The r2 pre-SynthStrip checks did not all pass.')
if not all(r2_gate.get('mask_checks', {}).values()):
    raise RuntimeError('The r2 mask checks did not all pass.')
if r2_gate.get('target_used_to_select_crop') is not False:
    raise RuntimeError('The r2 gate reports target use during crop selection.')
if r2_gate.get('bulk_atlas_registration_authorized') is not False:
    raise RuntimeError('Registration was authorized before the freeze audit.')

cow_audit = pd.read_csv(R1_COW_AUDIT_PATH)
if len(cow_audit) != EXPECTED_SUBJECTS:
    raise RuntimeError('The r1 CoW compatibility audit is incomplete.')
active_cow_audit = cow_audit.loc[
    cow_audit['subject_id'].astype(str) != CORRECTED_SUBJECT_ID
]
active_gross_mismatches = active_cow_audit.loc[
    active_cow_audit['cow_compatibility_category'].astype(str)
    == 'gross_mismatch'
]
if len(active_gross_mismatches):
    raise RuntimeError(
        'An active CoW prior still has a gross mismatch: '
        + ', '.join(active_gross_mismatches['subject_id'].astype(str))
    )

print('Trial8 r2 freeze and final integrity audit')
print(f'Candidate subjects: {len(candidate_manifest)}')
print(f'Split counts: {candidate_manifest["dev_split"].value_counts().to_dict()}')
print(f'Corrected subject: {CORRECTED_SUBJECT_ID}')
print(f'Disabled CoW priors: {disabled_cow_subjects}')
print(f'Active CoW gross mismatches: {len(active_gross_mismatches)}')
print('Manual visual QA approved: True')
print('Target arrays read: False')
print('Study trainable parameters: 0 (dataset freeze audit only)')
print('Bulk atlas registration authorized: False during audit')

case_rows = []
error_rows = []
run_start = time.time()

for case_index, row in enumerate(
    candidate_manifest.sort_values('subject_id').itertuples(index=False),
    start=1,
):
    subject_id = str(row.subject_id)
    dev_split = str(row.dev_split)
    npz_path = Path(row.vascufate_v2_npz_path)
    cow_prior_valid = as_bool(row.cow_prior_valid)
    is_corrected = subject_id == CORRECTED_SUBJECT_ID

    if is_corrected:
        mask_path = R2_MASK_PATH
        expected_npz_sha256 = str(r2_correction['corrected_npz_sha256'])
        expected_mask_sha256 = str(r2_gate['mask_sha256'])
        expected_mask_source = 'r2_corrected_sub0016'
        completion_record_path = R2_CANDIDATE_GATE_PATH
    else:
        case_dir = R1_CASE_ROOT / subject_id
        mask_path = case_dir / 'ncct_synthstrip_mask.nii.gz'
        completion_record_path = case_dir / 'complete.json'
        expected_mask_source = 'r1_bulk_verified'

    try:
        for path in [npz_path, mask_path, completion_record_path]:
            if not path.exists():
                raise FileNotFoundError(path)

        if not is_corrected:
            completion = json.loads(
                completion_record_path.read_text(encoding='utf-8')
            )
            if completion.get('subject_id') != subject_id:
                raise RuntimeError('r1 completion-record subject mismatch.')
            if completion.get('completed') is not True:
                raise RuntimeError('r1 completion record is incomplete.')
            if completion.get('target_arrays_read') is not False:
                raise RuntimeError('r1 completion record reports target access.')
            expected_npz_sha256 = str(completion['npz_sha256'])
            expected_mask_sha256 = str(completion['mask_sha256'])

        observed_npz_sha256 = file_sha256(npz_path)
        observed_mask_sha256 = file_sha256(mask_path)
        npz_hash_matches = observed_npz_sha256 == expected_npz_sha256
        mask_hash_matches = observed_mask_sha256 == expected_mask_sha256

        with np.load(npz_path, allow_pickle=False) as data:
            required_keys = {
                'image',
                'input_channels',
                'crop_mask',
                'cow_label',
                'cow_soft_union',
                'vessel_features_raw',
                'vessel_presence',
                'lvo_proximity',
                'affine',
                'subject_id',
            }
            missing_keys = sorted(required_keys - set(data.files))
            if missing_keys:
                raise RuntimeError(f'NPZ missing keys: {missing_keys}')
            if str(data['subject_id'].item()) != subject_id:
                raise RuntimeError('NPZ subject identifier mismatch.')

            image = data['image']
            input_channels = [str(value) for value in data['input_channels']]
            crop_mask = data['crop_mask'].astype(bool)
            cow_label = data['cow_label']
            cow_soft_union = data['cow_soft_union']
            vessel_features_raw = data['vessel_features_raw']
            vessel_presence = data['vessel_presence']
            lvo_proximity = data['lvo_proximity']
            affine = data['affine'].astype(np.float64)

            stored_cow_prior_valid = (
                bool(data['cow_prior_valid'].item())
                if 'cow_prior_valid' in data.files
                else True
            )

        mask_image, mask_array = effective_3d_mask(mask_path)
        brain_mask = mask_array > 0.5
        spatial_shape = tuple(image.shape[1:])
        spacing = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
        orientation = ''.join(nib.aff2axcodes(affine))

        labeled, component_count = ndimage.label(brain_mask)
        component_sizes = np.bincount(labeled.ravel())[1:]
        brain_voxels = int(brain_mask.sum())
        largest_component_fraction = (
            float(component_sizes.max() / brain_voxels)
            if len(component_sizes) and brain_voxels
            else 0.0
        )
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        brain_volume_ml = float(brain_voxels * voxel_volume_ml)
        crop_containment = float(
            np.logical_and(brain_mask, crop_mask).sum()
            / max(brain_voxels, 1)
        )
        edge_touch_x = bool(brain_mask[0].any() or brain_mask[-1].any())
        edge_touch_y = bool(
            brain_mask[:, 0].any() or brain_mask[:, -1].any()
        )

        corrected_cow_policy_passed = bool(
            (not is_corrected)
            or (
                not cow_prior_valid
                and not stored_cow_prior_valid
                and np.count_nonzero(cow_label) == 0
                and np.count_nonzero(cow_soft_union) == 0
                and np.count_nonzero(vessel_features_raw) == 0
                and np.count_nonzero(vessel_presence) == 0
            )
        )
        unchanged_cow_policy_passed = bool(
            is_corrected
            or (cow_prior_valid and stored_cow_prior_valid)
        )

        checks = {
            'npz_hash_matches_frozen_provenance': npz_hash_matches,
            'mask_hash_matches_frozen_provenance': mask_hash_matches,
            'image_has_15_channels': image.shape[0] == EXPECTED_IMAGE_CHANNELS,
            'channel_schema_has_15_names': (
                len(input_channels) == EXPECTED_IMAGE_CHANNELS
            ),
            'spatial_shapes_match': bool(
                spatial_shape
                == crop_mask.shape
                == cow_label.shape
                == cow_soft_union.shape
                == lvo_proximity.shape
                == brain_mask.shape
            ),
            'mask_affine_matches_npz': bool(
                np.allclose(mask_image.affine, affine, atol=1e-5, rtol=1e-5)
            ),
            'orientation_is_ras': orientation == 'RAS',
            'spacing_is_0p8_0p8_2p0': bool(
                np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0)
            ),
            'mask_is_finite': bool(np.isfinite(mask_array).all()),
            'mask_is_binary': bool(
                set(np.unique(mask_array).tolist()).issubset(
                    {0, 1, 0.0, 1.0}
                )
            ),
            'mask_is_nonempty': brain_voxels > 0,
            'brain_volume_is_plausible': bool(
                MIN_BRAIN_VOLUME_ML
                <= brain_volume_ml
                <= MAX_BRAIN_VOLUME_ML
            ),
            'largest_component_fraction_at_least_0p995': bool(
                largest_component_fraction
                >= MIN_LARGEST_COMPONENT_FRACTION
            ),
            'brain_mask_at_least_0p99_inside_crop_mask': bool(
                crop_containment >= MIN_CROP_CONTAINMENT
            ),
            'brain_does_not_touch_x_or_y_edges': bool(
                not (edge_touch_x or edge_touch_y)
            ),
            'corrected_subject_cow_policy_passed': (
                corrected_cow_policy_passed
            ),
            'unchanged_subject_cow_policy_passed': (
                unchanged_cow_policy_passed
            ),
        }
        integrity_passed = bool(all(checks.values()))

        case_rows.append({
            'subject_id': subject_id,
            'dev_split': dev_split,
            'artifact_revision': (
                CORRECTED_PREPROCESSING_VERSION
                if is_corrected
                else PARENT_PREPROCESSING_VERSION
            ),
            'npz_path': str(npz_path),
            'npz_sha256': observed_npz_sha256,
            'mask_path': str(mask_path),
            'mask_sha256': observed_mask_sha256,
            'mask_source': expected_mask_source,
            'cow_prior_valid': cow_prior_valid,
            'npz_hash_matches': npz_hash_matches,
            'mask_hash_matches': mask_hash_matches,
            'orientation': orientation,
            'spacing_x_mm': float(spacing[0]),
            'spacing_y_mm': float(spacing[1]),
            'spacing_z_mm': float(spacing[2]),
            'image_shape': str(tuple(image.shape)),
            'mask_shape': str(tuple(brain_mask.shape)),
            'brain_volume_ml': brain_volume_ml,
            'component_count': int(component_count),
            'largest_component_fraction': largest_component_fraction,
            'crop_containment_fraction': crop_containment,
            'corrected_cow_policy_passed': corrected_cow_policy_passed,
            'unchanged_cow_policy_passed': unchanged_cow_policy_passed,
            'integrity_passed': integrity_passed,
            'target_arrays_read': False,
        })

        if not integrity_passed:
            failed_checks = [
                key for key, value in checks.items() if not value
            ]
            error_rows.append({
                'subject_id': subject_id,
                'error': 'Failed checks: ' + ', '.join(failed_checks),
            })

        if case_index == 1 or case_index % 10 == 0 or case_index == 148:
            print(
                f'Checked {case_index}/{EXPECTED_SUBJECTS}: {subject_id}, '
                f'integrity={integrity_passed}'
            )

    except Exception as error:
        error_rows.append({
            'subject_id': subject_id,
            'error': f'{type(error).__name__}: {error}',
        })
        print(
            f'Failed {case_index}/{EXPECTED_SUBJECTS}: '
            f'{subject_id}: {error}'
        )

case_df = pd.DataFrame(case_rows).sort_values('subject_id').reset_index(
    drop=True
)
error_df = pd.DataFrame(error_rows)
atomic_write_csv(case_df, CASE_AUDIT_PATH)
if len(error_df):
    atomic_write_csv(error_df, ERROR_PATH)

all_case_integrity_passed = bool(
    len(case_df) == EXPECTED_SUBJECTS
    and case_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    and case_df['integrity_passed'].map(as_bool).all()
    and not len(error_df)
)

freeze_checks = {
    'manual_visual_qa_approved': MANUAL_VISUAL_QA_APPROVED,
    'candidate_has_148_subjects': len(candidate_manifest) == EXPECTED_SUBJECTS,
    'split_counts_preserved': (
        candidate_manifest['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
    'exactly_one_npz_revision': changed_subjects == [CORRECTED_SUBJECT_ID],
    'exactly_one_disabled_cow_prior': (
        disabled_cow_subjects == [CORRECTED_SUBJECT_ID]
    ),
    'no_active_cow_gross_mismatches': len(active_gross_mismatches) == 0,
    'r1_structural_verification_passed': (
        r1_summary.get('all_structural_masks_passed') is True
    ),
    'r2_correction_commit_checks_passed': all(
        r2_correction.get('commit_checks', {}).values()
    ),
    'r2_corrected_mask_checks_passed': bool(
        r2_gate.get('automated_safety_passed') is True
        and all(r2_gate.get('mask_checks', {}).values())
    ),
    'all_148_case_integrity_checks_passed': all_case_integrity_passed,
    'target_arrays_not_read_during_freeze_audit': True,
}

if not all(freeze_checks.values()):
    failed = [key for key, value in freeze_checks.items() if not value]
    print('\nFreeze checks:')
    display(pd.DataFrame([
        {'check': key, 'passed': value}
        for key, value in freeze_checks.items()
    ]))
    if len(error_df):
        print('\nCase integrity errors:')
        display(error_df)
    raise RuntimeError(
        'TRIAL8 R2 FREEZE AUDIT FAILED. The candidate manifest was not '
        f'frozen and registration remains locked. Failed checks: {failed}'
    )

frozen_manifest = candidate_manifest.copy()
case_lookup = case_df.set_index('subject_id')
frozen_manifest['trial8_npz_sha256'] = frozen_manifest['subject_id'].map(
    case_lookup['npz_sha256']
)
frozen_manifest['trial8_synthstrip_mask_path'] = (
    frozen_manifest['subject_id'].map(case_lookup['mask_path'])
)
frozen_manifest['trial8_synthstrip_mask_sha256'] = (
    frozen_manifest['subject_id'].map(case_lookup['mask_sha256'])
)
frozen_manifest['trial8_synthstrip_mask_source'] = (
    frozen_manifest['subject_id'].map(case_lookup['mask_source'])
)
frozen_manifest['trial8_manual_visual_qa_approved'] = True
frozen_manifest['trial8_dataset_frozen'] = True
frozen_manifest['trial8_bulk_atlas_registration_authorized'] = True
frozen_manifest['trial8_freeze_version'] = FREEZE_VERSION

if frozen_manifest[
    [
        'trial8_npz_sha256',
        'trial8_synthstrip_mask_path',
        'trial8_synthstrip_mask_sha256',
    ]
].isna().any().any():
    raise RuntimeError('Frozen manifest contains missing artifact metadata.')

atomic_write_csv(frozen_manifest, FROZEN_MANIFEST_PATH)
frozen_manifest_sha256 = file_sha256(FROZEN_MANIFEST_PATH)

committed_manifest = pd.read_csv(FROZEN_MANIFEST_PATH)
commit_checks = {
    'frozen_manifest_exists': FROZEN_MANIFEST_PATH.exists(),
    'frozen_manifest_has_148_subjects': (
        len(committed_manifest) == EXPECTED_SUBJECTS
    ),
    'frozen_manifest_subjects_are_unique': bool(
        not committed_manifest['subject_id'].duplicated().any()
    ),
    'frozen_manifest_split_counts_preserved': bool(
        committed_manifest['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
    'frozen_manifest_has_one_disabled_cow_prior': bool(
        sorted(
            committed_manifest.loc[
                ~committed_manifest['cow_prior_valid'].map(as_bool),
                'subject_id',
            ].astype(str).tolist()
        )
        == [CORRECTED_SUBJECT_ID]
    ),
    'frozen_manifest_registration_authorized': bool(
        committed_manifest[
            'trial8_bulk_atlas_registration_authorized'
        ].map(as_bool).all()
    ),
}
if not all(commit_checks.values()):
    failed = [key for key, value in commit_checks.items() if not value]
    raise RuntimeError(
        'Frozen manifest commit verification failed. '
        f'Failed checks: {failed}'
    )

freeze_gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'freeze_version': FREEZE_VERSION,
    'parent_preprocessing_version': PARENT_PREPROCESSING_VERSION,
    'corrected_preprocessing_version': CORRECTED_PREPROCESSING_VERSION,
    'subjects_frozen': EXPECTED_SUBJECTS,
    'unchanged_r1_subjects': 147,
    'corrected_r2_subjects': [CORRECTED_SUBJECT_ID],
    'disabled_cow_prior_subjects': disabled_cow_subjects,
    'active_cow_gross_mismatch_subjects': [],
    'manual_visual_qa_approved': True,
    'manual_visual_qa_note': MANUAL_VISUAL_QA_NOTE,
    'freeze_checks': freeze_checks,
    'commit_checks': commit_checks,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'dataset_frozen': True,
    'bulk_atlas_registration_authorized': True,
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'frozen_manifest_sha256': frozen_manifest_sha256,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH),
    'candidate_manifest_sha256': file_sha256(CANDIDATE_MANIFEST_PATH),
    'parent_manifest_path': str(PARENT_MANIFEST_PATH),
    'parent_manifest_sha256': file_sha256(PARENT_MANIFEST_PATH),
    'r1_verification_summary_path': str(R1_VERIFICATION_SUMMARY_PATH),
    'r1_verification_summary_sha256': file_sha256(
        R1_VERIFICATION_SUMMARY_PATH
    ),
    'r2_correction_audit_path': str(R2_CORRECTION_AUDIT_PATH),
    'r2_correction_audit_sha256': file_sha256(R2_CORRECTION_AUDIT_PATH),
    'r2_candidate_gate_path': str(R2_CANDIDATE_GATE_PATH),
    'r2_candidate_gate_sha256': file_sha256(R2_CANDIDATE_GATE_PATH),
    'case_audit_path': str(CASE_AUDIT_PATH),
    'case_audit_sha256': file_sha256(CASE_AUDIT_PATH),
    'elapsed_seconds': float(time.time() - run_start),
    'next_required_action': (
        'run bulk SyNRA atlas registration from the frozen r2 manifest'
    ),
}
atomic_write_json(freeze_gate, FREEZE_GATE_PATH)

print('\nFreeze checks:')
display(pd.DataFrame([
    {'check': key, 'passed': value}
    for key, value in freeze_checks.items()
]))

print('\nCommitted manifest checks:')
display(pd.DataFrame([
    {'check': key, 'passed': value}
    for key, value in commit_checks.items()
]))

print('\nFinal artifact summary:')
display(
    case_df.groupby(
        ['artifact_revision', 'mask_source', 'cow_prior_valid'],
        as_index=False,
    ).agg(
        subjects=('subject_id', 'count'),
        minimum_brain_volume_ml=('brain_volume_ml', 'min'),
        maximum_brain_volume_ml=('brain_volume_ml', 'max'),
        minimum_crop_containment=('crop_containment_fraction', 'min'),
    )
)

print('\n' + '=' * 80)
print('TRIAL8 R2 DATASET FREEZE AND INTEGRITY AUDIT PASSED')
print('=' * 80)
print(f'Frozen subjects:                     {EXPECTED_SUBJECTS}')
print('Unchanged r1 subjects:               147')
print(f'Corrected r2 subjects:               {[CORRECTED_SUBJECT_ID]}')
print(f'Disabled CoW priors:                 {disabled_cow_subjects}')
print('Active CoW gross mismatches:         0')
print('Manual visual QA approved:           True')
print('Target arrays read:                  False')
print('Study trainable parameters:          0 (dataset freeze audit only)')
print('Dataset frozen:                      True')
print('Bulk atlas registration authorized:  True')
print(f'Frozen manifest:                     {FROZEN_MANIFEST_PATH}')
print(f'Frozen manifest SHA-256:             {frozen_manifest_sha256}')
print(f'Case integrity audit:                {CASE_AUDIT_PATH}')
print(f'Freeze gate:                         {FREEZE_GATE_PATH}')
print('\nNext: run bulk SyNRA atlas registration from the frozen r2 manifest.')


In [ ]:
import os
import sys
import json
import time
import signal
import subprocess
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
ENVIRONMENT_RECORD_PATH = (
    PROJECT_ROOT / 'configs' / 'trial8_registration_environment.json'
)

PINNED_ANTSPYX = '0.6.3'
PINNED_SCIPY = '1.16.3'
NUMPY_SPEC = 'numpy<2.4'


def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None


def run_clean_import_smoke_test():
    smoke_code = r'''
import json
import numpy as np
import scipy
from scipy import ndimage
import ants

fixed = ants.from_numpy(np.zeros((8, 8, 8), dtype=np.float32))
moving = ants.from_numpy(np.zeros((8, 8, 8), dtype=np.float32))

print(json.dumps({
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "antspyx": getattr(ants, "__version__", "unknown"),
    "ants_dimension": int(fixed.dimension),
    "ndimage_available": callable(ndimage.binary_fill_holes),
}))
'''
    result = subprocess.run(
        [sys.executable, '-c', smoke_code],
        text=True,
        capture_output=True,
    )

    if result.returncode != 0:
        return False, None, result.stderr.strip()

    output_lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not output_lines:
        return False, None, 'Clean import smoke test returned no output.'

    try:
        versions = json.loads(output_lines[-1])
    except json.JSONDecodeError as exc:
        return False, None, f'Could not parse smoke-test output: {exc}'

    version_ok = (
        versions.get('scipy') == PINNED_SCIPY
        and versions.get('antspyx') == PINNED_ANTSPYX
        and versions.get('ndimage_available') is True
        and versions.get('ants_dimension') == 3
    )

    if not version_ok:
        return False, versions, 'Installed versions do not match the Trial8 pins.'

    return True, versions, None


def install_registration_stack():
    dependency_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--upgrade',
        '--no-cache-dir',
        NUMPY_SPEC,
        'pandas',
        'nibabel',
        'pyyaml',
        'statsmodels',
        'webcolors',
        'matplotlib',
        'Pillow',
        'requests',
        'scikit-learn',
        'packaging',
    ]
    subprocess.check_call(dependency_command)

    
    
    pinned_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--force-reinstall',
        '--no-deps',
        '--no-cache-dir',
        f'scipy=={PINNED_SCIPY}',
        f'antspyx=={PINNED_ANTSPYX}',
    ]
    subprocess.check_call(pinned_command)


metadata_versions = {
    'antspyx': installed_version('antspyx'),
    'scipy': installed_version('scipy'),
    'numpy': installed_version('numpy'),
}

loaded_sensitive_roots = sorted({
    module_name.split('.')[0]
    for module_name in sys.modules
    if module_name.split('.')[0] in {'scipy', 'ants', 'statsmodels'}
})

smoke_passed, smoke_versions, smoke_error = run_clean_import_smoke_test()
installation_performed = False

if not smoke_passed:
    print('Trial8 registration environment needs repair.')
    print(f'Installed metadata before repair: {metadata_versions}')
    if smoke_versions is not None:
        print(f'Clean-process versions before repair: {smoke_versions}')
    if smoke_error:
        print(f'Clean-process error: {smoke_error}')

    install_registration_stack()
    installation_performed = True

    smoke_passed, smoke_versions, smoke_error = run_clean_import_smoke_test()
    if not smoke_passed:
        raise RuntimeError(
            'Trial8 registration environment repair failed. '
            f'Clean-process error: {smoke_error}'
        )

ENVIRONMENT_RECORD_PATH.parent.mkdir(parents=True, exist_ok=True)
environment_record = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version,
    'python_executable': sys.executable,
    'pinned_antspyx': PINNED_ANTSPYX,
    'pinned_scipy': PINNED_SCIPY,
    'numpy_specification': NUMPY_SPEC,
    'clean_process_versions': smoke_versions,
    'clean_import_smoke_test_passed': bool(smoke_passed),
    'installation_performed': bool(installation_performed),
}
ENVIRONMENT_RECORD_PATH.write_text(
    json.dumps(environment_record, indent=2),
    encoding='utf-8',
)

restart_required = installation_performed or bool(loaded_sensitive_roots)

if restart_required:
    print('\nTrial8 ANTs/SciPy files are now consistent on disk.')
    if loaded_sensitive_roots:
        print(f'Loaded modules requiring a clean restart: {loaded_sensitive_roots}')
    print(f'Verified clean-process versions: {smoke_versions}')
    print(f'Saved environment record: {ENVIRONMENT_RECORD_PATH}')
    print('Colab will restart now to clear stale imported modules.')
    print('After reconnecting, run this environment cell once more.')
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), signal.SIGKILL)

print('\n' + '=' * 80)
print('TRIAL8 REGISTRATION ENVIRONMENT READY')
print('=' * 80)
print(f'Python:   {sys.version.split()[0]}')
print(f'NumPy:    {smoke_versions["numpy"]}')
print(f'SciPy:    {smoke_versions["scipy"]}')
print(f'ANTsPyX:  {smoke_versions["antspyx"]}')
print(f'Record:   {ENVIRONMENT_RECORD_PATH}')
print('\nNext: run the standalone Trial8 registration preflight cell.')


In [ ]:
import os
import sys
import json
import time
import shutil
import hashlib
import tempfile
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path



ITK_THREADS = min(4, os.cpu_count() or 2)
os.environ['ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS'] = str(ITK_THREADS)

PINNED_ANTSPYX = '0.6.3'
PINNED_SCIPY = '1.16.3'
BULK_CODE_REVISION = 'r1_frozen_r2_synra_all148'


def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None


dependency_versions = {
    'antspyx': installed_version('antspyx'),
    'scipy': installed_version('scipy'),
}
dependency_errors = []
if dependency_versions['antspyx'] != PINNED_ANTSPYX:
    dependency_errors.append(
        f'antspyx must be {PINNED_ANTSPYX}; '
        f'found {dependency_versions["antspyx"]}'
    )
if dependency_versions['scipy'] != PINNED_SCIPY:
    dependency_errors.append(
        f'scipy must be {PINNED_SCIPY}; '
        f'found {dependency_versions["scipy"]}'
    )
if dependency_errors:
    raise RuntimeError(
        'Trial8 registration environment is not ready. Run the standalone '
        'trial8_registration_environment_setup.py cell, allow its requested '
        'runtime restart, run that setup cell once more, and then rerun this '
        'bulk cell. Problems: ' + '; '.join(dependency_errors)
    )

try:
    import numpy as np
    import pandas as pd
    import nibabel as nib
    import scipy
    from scipy import ndimage
    import matplotlib.pyplot as plt
    import ants
except Exception as exc:
    raise RuntimeError(
        'The pinned Trial8 registration stack could not be imported. Run the '
        'standalone registration-environment setup cell before this block.'
    ) from exc

from google.colab import drive
from IPython.display import display


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PARENT_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
FROZEN_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
SCHEMA_VERSION = 'vascular_hierarchy_v1'
REGISTRATION_VERSION = 'v1_jhu_to_ncct_synra_synthstrip_r2_all148'
SYNTHSTRIP_VERSION = 'synthstrip_v1_default_csf'
SEED = 20240824

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_frozen.csv'
)
FREEZE_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / FROZEN_PREPROCESSING_VERSION
    / 'final_verification'
    / 'v1_r2_mixed_147r1_1r2'
    / 'r2_dataset_freeze_gate.json'
)

ATLAS_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PARENT_PREPROCESSING_VERSION
    / 'vascular_atlas'
)
ACQUISITION_GATE_PATH = (
    ATLAS_ROOT / 'acquisition_audit' / 'atlas_acquisition_gate.json'
)
SCHEMA_DIR = ATLAS_ROOT / 'schema' / SCHEMA_VERSION
SCHEMA_GATE_PATH = SCHEMA_DIR / 'schema_freeze_gate.json'
SCHEMA_JSON_PATH = SCHEMA_DIR / 'vascular_label_schema.json'
SCHEMA_CSV_PATH = SCHEMA_DIR / 'vascular_label_schema.csv'
LABEL_AUDIT_PATH = (
    ATLAS_ROOT
    / 'label_hierarchy_audit'
    / 'atlas_label_hierarchy_audit.json'
)
TRANSFORM_SELECTION_GATE_PATH = (
    ATLAS_ROOT
    / 'registration_preflight'
    / 'v2_jhu_to_ncct_synra_synthstrip_mi'
    / 'transform_selection_gate.json'
)
JHU_REFERENCE_PATH = (
    ATLAS_ROOT
    / 'reference'
    / 'JHU_MNI_SS'
    / 'JHU_MNI_SS_T1_brain_182x218x182.nii.gz'
)
FULL_FINE_ATLAS_PATH = (
    ATLAS_ROOT
    / 'source'
    / 'Arterial_Atlas'
    / 'data'
    / 'Atlas_182'
    / 'ArterialAtlas.nii'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / FROZEN_PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'bulk_registration'
    / REGISTRATION_VERSION
)
CASE_ROOT = OUTPUT_ROOT / 'cases'
QUARANTINE_ROOT = OUTPUT_ROOT / 'incomplete_quarantine'
CONFIG_PATH = OUTPUT_ROOT / 'bulk_registration_config.json'
AUDIT_PATH = OUTPUT_ROOT / 'bulk_registration_audit.csv'
FAILURE_PATH = OUTPUT_ROOT / 'bulk_registration_failures.csv'
SUMMARY_PATH = OUTPUT_ROOT / 'bulk_registration_summary.json'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)

REGISTRATION_CONFIG = {
    'type_of_transform': 'SyNRA',
    'aff_metric': 'mattes',
    'aff_sampling': 32,
    'aff_random_sampling_rate': 0.25,
    'syn_metric': 'mattes',
    'syn_sampling': 32,
    'reg_iterations': [80, 60, 40, 20],
    'grad_step': 0.15,
    'flow_sigma': 3,
    'total_sigma': 0,
    'mask_all_stages': True,
    'random_seed': SEED,
    'smoothing_in_mm': True,
    'itk_threads': ITK_THREADS,
}


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def object_sha256(payload):
    encoded = json.dumps(
        payload,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


def atomic_write_json(payload, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temporary.replace(path)


def atomic_write_csv(frame, path):
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() == 'true'


def save_nifti(data, affine, path, dtype):
    array = np.asarray(data, dtype=dtype)
    image = nib.Nifti1Image(array, np.asarray(affine, dtype=np.float64))
    image.set_sform(np.asarray(affine, dtype=np.float64), code=2)
    image.set_qform(np.asarray(affine, dtype=np.float64), code=1)
    nib.save(image, str(path))


def dice_binary(first, second, eps=1e-6):
    first = np.asarray(first, dtype=bool)
    second = np.asarray(second, dtype=bool)
    return float(
        (2.0 * np.logical_and(first, second).sum() + eps)
        / (first.sum() + second.sum() + eps)
    )


def normalized_mutual_information(first, second, mask, bins=64):
    first = np.asarray(first, dtype=np.float32)
    second = np.asarray(second, dtype=np.float32)
    mask = (
        np.asarray(mask, dtype=bool)
        & np.isfinite(first)
        & np.isfinite(second)
    )
    if mask.sum() < 100:
        return float('nan')
    x = first[mask]
    y = second[mask]
    histogram, _, _ = np.histogram2d(x, y, bins=bins)
    probability = histogram / max(histogram.sum(), 1.0)
    px = probability.sum(axis=1)
    py = probability.sum(axis=0)
    nz_joint = probability > 0
    nz_x = px > 0
    nz_y = py > 0
    h_joint = -float(
        np.sum(probability[nz_joint] * np.log(probability[nz_joint]))
    )
    h_x = -float(np.sum(px[nz_x] * np.log(px[nz_x])))
    h_y = -float(np.sum(py[nz_y] * np.log(py[nz_y])))
    return float((h_x + h_y) / h_joint) if h_joint > 0 else float('nan')


def center_of_mass_world(mask, affine):
    coordinates = np.argwhere(np.asarray(mask, dtype=bool))
    if not len(coordinates):
        return np.full(3, np.nan, dtype=np.float64)
    center_voxel = coordinates.mean(axis=0)
    return np.asarray(
        nib.affines.apply_affine(affine, center_voxel),
        dtype=np.float64,
    )


def homolog_pair_metrics(labels, affine):
    present_pairs = 0
    correctly_ordered_pairs = 0
    x_differences = []
    for left_label in range(1, 30, 2):
        right_label = left_label + 1
        left_mask = labels == left_label
        right_mask = labels == right_label
        if left_mask.any() and right_mask.any():
            present_pairs += 1
            left_x = center_of_mass_world(left_mask, affine)[0]
            right_x = center_of_mass_world(right_mask, affine)[0]
            difference = float(right_x - left_x)
            x_differences.append(difference)
            if difference > 0:
                correctly_ordered_pairs += 1
    return present_pairs, correctly_ordered_pairs, x_differences


def label_boundary(labels):
    labels = np.asarray(labels)
    boundary = np.zeros(labels.shape, dtype=bool)
    for axis in range(3):
        lower = [slice(None)] * 3
        upper = [slice(None)] * 3
        lower[axis] = slice(0, -1)
        upper[axis] = slice(1, None)
        difference = labels[tuple(lower)] != labels[tuple(upper)]
        boundary[tuple(lower)] |= difference
        boundary[tuple(upper)] |= difference
    return boundary & (labels > 0)


def display_slice(array, axis, index):
    if axis == 0:
        image = array[index, :, :]
    elif axis == 1:
        image = array[:, index, :]
    else:
        image = array[:, :, index]
    return np.rot90(image)


def save_visual_qa(
    fixed_array,
    nonlinear_warped,
    syn_fine_clipped,
    cow_label,
    lvo_proximity,
    fixed_mask,
    output_path,
    subject_id,
):
    center = np.rint(np.argwhere(fixed_mask).mean(axis=0)).astype(int)
    axes = [2, 1, 0]
    axis_names = ['axial', 'coronal', 'sagittal']
    indices = [int(center[axis]) for axis in axes]
    mask_boundary = fixed_mask & ~ndimage.binary_erosion(fixed_mask)
    syn_boundaries = label_boundary(syn_fine_clipped)
    syn_ventricles = np.isin(syn_fine_clipped, [31, 32])
    vessel_mask = cow_label > 0
    lvo_mask = lvo_proximity >= 0.5

    figure, plot_axes = plt.subplots(3, 3, figsize=(15, 13))
    for row, (axis, axis_name, index) in enumerate(
        zip(axes, axis_names, indices)
    ):
        fixed_slice = display_slice(fixed_array, axis, index)
        warped_slice = display_slice(nonlinear_warped, axis, index)
        mask_slice = display_slice(mask_boundary, axis, index)
        labels_slice = display_slice(syn_fine_clipped, axis, index)
        boundary_slice = display_slice(syn_boundaries, axis, index)
        ventricle_slice = display_slice(syn_ventricles, axis, index)
        vessel_slice = display_slice(vessel_mask, axis, index)
        lvo_slice = display_slice(lvo_mask, axis, index)

        plot_axes[row, 0].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 0].imshow(
            np.ma.masked_where(~mask_slice, mask_slice),
            cmap='autumn',
            alpha=0.95,
        )
        plot_axes[row, 0].set_title(f'{axis_name}: NCCT + brain mask')

        plot_axes[row, 1].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 1].imshow(warped_slice, cmap='magma', alpha=0.40)
        plot_axes[row, 1].set_title('SyN JHU overlay')

        plot_axes[row, 2].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 2].imshow(
            np.ma.masked_where(labels_slice == 0, labels_slice),
            cmap='turbo',
            alpha=0.28,
            vmin=1,
            vmax=32,
        )
        plot_axes[row, 2].imshow(
            np.ma.masked_where(~boundary_slice, boundary_slice),
            cmap='cool',
            alpha=0.85,
        )
        if ventricle_slice.any():
            plot_axes[row, 2].contour(
                ventricle_slice,
                levels=[0.5],
                colors='lime',
                linewidths=0.8,
            )
        if vessel_slice.any():
            plot_axes[row, 2].contour(
                vessel_slice,
                levels=[0.5],
                colors='magenta',
                linewidths=0.7,
            )
        if lvo_slice.any():
            plot_axes[row, 2].contour(
                lvo_slice,
                levels=[0.5],
                colors='yellow',
                linewidths=1.0,
            )
        plot_axes[row, 2].set_title('SyN territories clipped')

        for column in range(3):
            plot_axes[row, column].axis('off')

    figure.suptitle(
        f'{subject_id} bulk SyNRA QA\n'
        'orange=brain, cyan=territories, lime=ventricles, '
        'magenta=CoW, yellow=LVO',
        fontsize=14,
    )
    figure.tight_layout(rect=[0, 0, 1, 0.95])
    figure.savefig(output_path, dpi=125, bbox_inches='tight')
    plt.close(figure)


def copy_transform_list(transform_paths, destination, prefix):
    copied = []
    for index, source in enumerate(transform_paths):
        source = Path(source)
        suffix = ''.join(source.suffixes) or '.transform'
        target = destination / f'{prefix}_{index:02d}{suffix}'
        shutil.copy2(source, target)
        copied.append(target.name)
    return copied


def load_effective_3d(path, dtype):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 3:
        raise RuntimeError(f'Unsupported 3D image shape: {path}: {array.shape}')
    return image, np.asarray(array, dtype=dtype)


def quarantine_incomplete_directory(subject_dir):
    if not subject_dir.exists():
        return None
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    target = QUARANTINE_ROOT / f'{subject_dir.name}_{timestamp}'
    counter = 1
    while target.exists():
        target = QUARANTINE_ROOT / (
            f'{subject_dir.name}_{timestamp}_{counter:02d}'
        )
        counter += 1
    shutil.move(str(subject_dir), str(target))
    return target


required_paths = [
    FROZEN_MANIFEST_PATH,
    FREEZE_GATE_PATH,
    ACQUISITION_GATE_PATH,
    SCHEMA_GATE_PATH,
    SCHEMA_JSON_PATH,
    SCHEMA_CSV_PATH,
    LABEL_AUDIT_PATH,
    TRANSFORM_SELECTION_GATE_PATH,
    JHU_REFERENCE_PATH,
    FULL_FINE_ATLAS_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 bulk-registration prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

for path in [OUTPUT_ROOT, CASE_ROOT, QUARANTINE_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

freeze_gate = json.loads(FREEZE_GATE_PATH.read_text(encoding='utf-8'))
acquisition_gate = json.loads(
    ACQUISITION_GATE_PATH.read_text(encoding='utf-8')
)
schema_gate = json.loads(SCHEMA_GATE_PATH.read_text(encoding='utf-8'))
schema_record = json.loads(SCHEMA_JSON_PATH.read_text(encoding='utf-8'))
label_audit = json.loads(LABEL_AUDIT_PATH.read_text(encoding='utf-8'))
selection_gate = json.loads(
    TRANSFORM_SELECTION_GATE_PATH.read_text(encoding='utf-8')
)

if freeze_gate.get('dataset_frozen') is not True:
    raise RuntimeError('The r2 dataset is not frozen.')
if freeze_gate.get('bulk_atlas_registration_authorized') is not True:
    raise RuntimeError('The r2 freeze gate does not authorize registration.')
if not all(freeze_gate.get('freeze_checks', {}).values()):
    raise RuntimeError('The r2 freeze checks did not all pass.')
if file_sha256(FROZEN_MANIFEST_PATH) != freeze_gate.get(
    'frozen_manifest_sha256'
):
    raise RuntimeError('The frozen r2 manifest changed after its gate.')
if acquisition_gate.get('gate_passed') is not True:
    raise RuntimeError('The atlas acquisition gate is not passed.')
if schema_gate.get('gate_passed') is not True:
    raise RuntimeError('The atlas schema gate is not passed.')
if schema_record.get('schema_version') != SCHEMA_VERSION:
    raise RuntimeError('Unexpected vascular schema version.')
if file_sha256(SCHEMA_JSON_PATH) != schema_gate.get('schema_json_sha256'):
    raise RuntimeError('The vascular schema changed after its freeze gate.')
if file_sha256(FULL_FINE_ATLAS_PATH) != schema_record.get(
    'source_fine_atlas_sha256'
):
    raise RuntimeError('The fine arterial atlas changed after schema freeze.')
if file_sha256(JHU_REFERENCE_PATH) != label_audit.get(
    'jhu_reference_sha256'
):
    raise RuntimeError('The JHU reference changed after its geometry audit.')
if selection_gate.get('gate_passed') is not True:
    raise RuntimeError('The registration transform-selection gate is not passed.')
if selection_gate.get('selected_registration') != 'SyNRA':
    raise RuntimeError('The frozen registration policy is not SyNRA.')

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'cow_prior_valid',
    'trial8_npz_sha256',
    'trial8_synthstrip_mask_path',
    'trial8_synthstrip_mask_sha256',
    'trial8_bulk_atlas_registration_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen r2 manifest missing columns: {missing_columns}')
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(f'Frozen manifest has {len(manifest_df)} subjects.')
if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError('Frozen manifest contains duplicate subjects.')
if manifest_df['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
    raise RuntimeError('Frozen manifest split counts changed.')
if not manifest_df[
    'trial8_bulk_atlas_registration_authorized'
].map(as_bool).all():
    raise RuntimeError('A frozen manifest row does not authorize registration.')

schema_df = pd.read_csv(SCHEMA_CSV_PATH)
graph_node_mask = schema_df['use_as_graph_node'].map(as_bool)
vascular_schema_df = schema_df.loc[graph_node_mask].copy()
if set(vascular_schema_df['fine_label'].astype(int)) != set(range(1, 31)):
    raise RuntimeError('The graph-node schema is not fine labels 1 through 30.')
fine_to_level2 = dict(
    zip(
        schema_df['fine_label'].astype(int),
        schema_df['level2_label'].astype(int),
    )
)

jhu_image, jhu_array = load_effective_3d(JHU_REFERENCE_PATH, np.float32)
fine_image, fine_source_labels = load_effective_3d(
    FULL_FINE_ATLAS_PATH,
    np.int16,
)
fine_source_labels = np.rint(fine_source_labels).astype(np.int16)
if jhu_array.shape != fine_source_labels.shape:
    raise RuntimeError('JHU reference and fine atlas shapes differ.')
if not np.allclose(jhu_image.affine, fine_image.affine, atol=1e-5, rtol=1e-5):
    raise RuntimeError('JHU reference and fine atlas affines differ.')
moving_mask_np = jhu_array > 0

manifest_sha256 = file_sha256(FROZEN_MANIFEST_PATH)
registration_signature_payload = {
    'code_revision': BULK_CODE_REVISION,
    'registration_version': REGISTRATION_VERSION,
    'registration_config': REGISTRATION_CONFIG,
    'manifest_sha256': manifest_sha256,
    'freeze_gate_sha256': file_sha256(FREEZE_GATE_PATH),
    'selection_gate_sha256': file_sha256(TRANSFORM_SELECTION_GATE_PATH),
    'schema_json_sha256': file_sha256(SCHEMA_JSON_PATH),
    'jhu_reference_sha256': file_sha256(JHU_REFERENCE_PATH),
    'fine_atlas_sha256': file_sha256(FULL_FINE_ATLAS_PATH),
}
registration_signature = object_sha256(registration_signature_payload)

config_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'registration_version': REGISTRATION_VERSION,
    'bulk_code_revision': BULK_CODE_REVISION,
    'registration_signature': registration_signature,
    'registration_signature_payload': registration_signature_payload,
    'subjects': EXPECTED_SUBJECTS,
    'splits': EXPECTED_SPLITS,
    'fixed_image': '0.75 * ncct_brain_window + 0.25 * ncct_broad_window',
    'fixed_mask': 'frozen per-subject SynthStrip default-CSF mask',
    'moving_image': str(JHU_REFERENCE_PATH),
    'moving_labels': str(FULL_FINE_ATLAS_PATH),
    'registration': REGISTRATION_CONFIG,
    'final_label_policy': 'SyN labels clipped to frozen SynthStrip mask',
    'case_resumable': True,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(config_record, CONFIG_PATH)


def validate_completed_case(subject_dir, manifest_row):
    complete_path = subject_dir / 'complete.json'
    if not complete_path.exists():
        return None
    complete = json.loads(complete_path.read_text(encoding='utf-8'))
    expected_values = {
        'subject_id': str(manifest_row['subject_id']),
        'registration_version': REGISTRATION_VERSION,
        'bulk_code_revision': BULK_CODE_REVISION,
        'registration_signature': registration_signature,
        'npz_sha256': str(manifest_row['trial8_npz_sha256']),
        'synthstrip_mask_sha256': str(
            manifest_row['trial8_synthstrip_mask_sha256']
        ),
    }
    if any(complete.get(key) != value for key, value in expected_values.items()):
        return None
    if complete.get('completed') is not True:
        return None
    if complete.get('target_arrays_read') is not False:
        return None
    output_hashes = complete.get('output_sha256', {})
    if not output_hashes:
        return None
    for file_name, expected_hash in output_hashes.items():
        output_path = subject_dir / file_name
        if not output_path.exists():
            return None
        if file_sha256(output_path) != expected_hash:
            return None
    return complete


def process_subject(manifest_row):
    subject_id = str(manifest_row['subject_id'])
    dev_split = str(manifest_row['dev_split'])
    npz_path = Path(manifest_row['vascufate_v2_npz_path'])
    mask_path = Path(manifest_row['trial8_synthstrip_mask_path'])
    subject_dir = CASE_ROOT / subject_id

    if subject_dir.exists():
        completed = validate_completed_case(subject_dir, manifest_row)
        if completed is not None:
            print(f'Reused completed case: {subject_id}')
            return completed, True
        quarantined = quarantine_incomplete_directory(subject_dir)
        print(f'Quarantined incomplete case: {quarantined}')

    for path in [npz_path, mask_path]:
        if not path.exists():
            raise FileNotFoundError(path)
    observed_npz_sha256 = file_sha256(npz_path)
    observed_mask_sha256 = file_sha256(mask_path)
    if observed_npz_sha256 != str(manifest_row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id} NPZ hash changed after dataset freeze.')
    if observed_mask_sha256 != str(
        manifest_row['trial8_synthstrip_mask_sha256']
    ):
        raise RuntimeError(f'{subject_id} mask hash changed after dataset freeze.')

    local_parent = Path(
        tempfile.mkdtemp(prefix=f'trial8_bulk_reg_{subject_id}_', dir='/tmp')
    )
    local_case_dir = local_parent / subject_id
    local_case_dir.mkdir(parents=True, exist_ok=False)
    start_time = time.time()

    try:
        with np.load(npz_path, allow_pickle=False) as data:
            required_keys = {
                'image',
                'crop_mask',
                'cow_label',
                'lvo_proximity',
                'input_channels',
                'affine',
                'subject_id',
            }
            missing_keys = sorted(required_keys - set(data.files))
            if missing_keys:
                raise RuntimeError(f'{subject_id} NPZ missing keys: {missing_keys}')
            if str(data['subject_id'].item()) != subject_id:
                raise RuntimeError(f'{subject_id} NPZ subject mismatch.')
            channel_names = [str(value) for value in data['input_channels']]
            brain_index = channel_names.index('ncct_brain_window')
            broad_index = channel_names.index('ncct_broad_window')
            brain_window = data['image'][brain_index].astype(np.float32)
            broad_window = data['image'][broad_index].astype(np.float32)
            crop_mask_np = data['crop_mask'].astype(bool)
            cow_label = data['cow_label'].astype(np.uint8)
            lvo_proximity = data['lvo_proximity'].astype(np.float32)
            patient_affine = data['affine'].astype(np.float64)

        mask_image, mask_array = load_effective_3d(mask_path, np.float32)
        fixed_mask_np = mask_array > 0.5
        if not (
            brain_window.shape
            == broad_window.shape
            == crop_mask_np.shape
            == cow_label.shape
            == lvo_proximity.shape
            == fixed_mask_np.shape
        ):
            raise RuntimeError(f'{subject_id} spatial shapes differ.')
        if not np.allclose(
            mask_image.affine,
            patient_affine,
            atol=1e-5,
            rtol=1e-5,
        ):
            raise RuntimeError(f'{subject_id} mask affine differs from NPZ.')
        if ''.join(nib.aff2axcodes(patient_affine)) != 'RAS':
            raise RuntimeError(f'{subject_id} is not RAS-oriented.')
        spacing = np.sqrt(np.sum(patient_affine[:3, :3] ** 2, axis=0))
        if not np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0):
            raise RuntimeError(f'{subject_id} spacing changed.')
        if not fixed_mask_np.any():
            raise RuntimeError(f'{subject_id} fixed mask is empty.')

        brain_inside_crop_fraction = float(
            np.logical_and(fixed_mask_np, crop_mask_np).sum()
            / max(fixed_mask_np.sum(), 1)
        )
        fixed_array = np.clip(
            0.75 * brain_window + 0.25 * broad_window,
            0.0,
            1.0,
        ).astype(np.float32)
        fixed_array[~fixed_mask_np] = 0.0

        fixed_path = local_case_dir / 'patient_ncct_fixed.nii.gz'
        fixed_mask_path = local_case_dir / 'patient_registration_mask.nii.gz'
        moving_path = local_case_dir / 'jhu_mni_ss_moving.nii.gz'
        moving_mask_path = local_case_dir / 'jhu_mni_ss_mask.nii.gz'
        fine_path = local_case_dir / 'arterial_atlas_full_fine_3d.nii.gz'
        save_nifti(fixed_array, patient_affine, fixed_path, np.float32)
        save_nifti(
            fixed_mask_np.astype(np.uint8),
            patient_affine,
            fixed_mask_path,
            np.uint8,
        )
        save_nifti(jhu_array, jhu_image.affine, moving_path, np.float32)
        save_nifti(
            moving_mask_np.astype(np.uint8),
            jhu_image.affine,
            moving_mask_path,
            np.uint8,
        )
        save_nifti(
            fine_source_labels,
            fine_image.affine,
            fine_path,
            np.int16,
        )

        fixed = ants.image_read(str(fixed_path), pixeltype='float')
        fixed_mask = ants.image_read(
            str(fixed_mask_path),
            pixeltype='unsigned char',
        )
        moving = ants.image_read(str(moving_path), pixeltype='float')
        moving_mask = ants.image_read(
            str(moving_mask_path),
            pixeltype='unsigned char',
        )
        moving_labels = ants.image_read(str(fine_path), pixeltype='float')

        print(f'Registering {subject_id} with SyNRA + Mattes MI...')
        registration = ants.registration(
            fixed=fixed,
            moving=moving,
            type_of_transform=REGISTRATION_CONFIG['type_of_transform'],
            outprefix=str(local_case_dir / 'ants_'),
            mask=fixed_mask,
            moving_mask=moving_mask,
            mask_all_stages=REGISTRATION_CONFIG['mask_all_stages'],
            grad_step=REGISTRATION_CONFIG['grad_step'],
            flow_sigma=REGISTRATION_CONFIG['flow_sigma'],
            total_sigma=REGISTRATION_CONFIG['total_sigma'],
            aff_metric=REGISTRATION_CONFIG['aff_metric'],
            aff_sampling=REGISTRATION_CONFIG['aff_sampling'],
            aff_random_sampling_rate=(
                REGISTRATION_CONFIG['aff_random_sampling_rate']
            ),
            syn_metric=REGISTRATION_CONFIG['syn_metric'],
            syn_sampling=REGISTRATION_CONFIG['syn_sampling'],
            reg_iterations=tuple(REGISTRATION_CONFIG['reg_iterations']),
            random_seed=REGISTRATION_CONFIG['random_seed'],
            smoothing_in_mm=REGISTRATION_CONFIG['smoothing_in_mm'],
            verbose=False,
        )

        forward_transforms = list(registration['fwdtransforms'])
        inverse_transforms = list(registration['invtransforms'])
        warp_transforms = [
            path
            for path in forward_transforms
            if str(path).lower().endswith(('.nii', '.nii.gz'))
            and 'inversewarp' not in str(path).lower()
        ]
        if not warp_transforms:
            raise RuntimeError(f'{subject_id} produced no forward warp.')

        nonlinear_warped = registration['warpedmovout']
        nonlinear_mask = ants.apply_transforms(
            fixed=fixed,
            moving=moving_mask,
            transformlist=forward_transforms,
            interpolator='nearestNeighbor',
        )
        syn_labels_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving_labels,
            transformlist=forward_transforms,
            interpolator='genericLabel',
        )

        nonlinear_warped_np = nonlinear_warped.numpy().astype(np.float32)
        nonlinear_mask_np = nonlinear_mask.numpy() > 0.5
        syn_full_fine_raw = np.rint(
            syn_labels_image.numpy()
        ).astype(np.int16)
        unexpected_labels = sorted(
            set(int(value) for value in np.unique(syn_full_fine_raw))
            - set(range(33))
        )

        syn_full_fine_clipped = syn_full_fine_raw.copy()
        syn_full_fine_clipped[~fixed_mask_np] = 0
        syn_vascular_fine = syn_full_fine_clipped.copy()
        syn_vascular_fine[np.isin(syn_vascular_fine, [31, 32])] = 0
        syn_level2 = np.zeros_like(syn_full_fine_clipped, dtype=np.int16)
        for fine_label, parent_label in fine_to_level2.items():
            syn_level2[syn_full_fine_clipped == fine_label] = parent_label
        atlas_valid_mask = syn_full_fine_clipped > 0

        jacobian_image = ants.create_jacobian_determinant_image(
            fixed,
            warp_transforms[0],
            do_log=False,
            geom=False,
        )
        jacobian_np = jacobian_image.numpy().astype(np.float32)

        syn_fine_path = (
            local_case_dir / 'arterial_atlas_full_fine_syn_clipped.nii.gz'
        )
        syn_vascular_path = (
            local_case_dir / 'arterial_atlas_vascular30_syn_clipped.nii.gz'
        )
        syn_level2_path = (
            local_case_dir / 'arterial_atlas_level2_syn_clipped.nii.gz'
        )
        atlas_valid_path = local_case_dir / 'atlas_valid_mask.nii.gz'
        jacobian_path = local_case_dir / 'syn_jacobian_determinant.nii.gz'
        qa_path = local_case_dir / 'registration_visual_qa.png'

        save_nifti(
            syn_full_fine_clipped,
            patient_affine,
            syn_fine_path,
            np.int16,
        )
        save_nifti(
            syn_vascular_fine,
            patient_affine,
            syn_vascular_path,
            np.int16,
        )
        save_nifti(
            syn_level2,
            patient_affine,
            syn_level2_path,
            np.int16,
        )
        save_nifti(
            atlas_valid_mask.astype(np.uint8),
            patient_affine,
            atlas_valid_path,
            np.uint8,
        )
        save_nifti(jacobian_np, patient_affine, jacobian_path, np.float32)
        save_visual_qa(
            fixed_array=fixed_array,
            nonlinear_warped=nonlinear_warped_np,
            syn_fine_clipped=syn_full_fine_clipped,
            cow_label=cow_label,
            lvo_proximity=lvo_proximity,
            fixed_mask=fixed_mask_np,
            output_path=qa_path,
            subject_id=subject_id,
        )

        copied_forward = copy_transform_list(
            forward_transforms,
            local_case_dir,
            'forward_transform',
        )
        copied_inverse = copy_transform_list(
            inverse_transforms,
            local_case_dir,
            'inverse_transform',
        )

        
        
        for intermediate_path in [
            fixed_path,
            fixed_mask_path,
            moving_path,
            moving_mask_path,
            fine_path,
        ]:
            if intermediate_path.exists():
                intermediate_path.unlink()
        for intermediate_path in local_case_dir.glob('ants_*'):
            if intermediate_path.is_file():
                intermediate_path.unlink()

        fixed_center = center_of_mass_world(fixed_mask_np, patient_affine)
        nonlinear_center = center_of_mass_world(
            nonlinear_mask_np,
            patient_affine,
        )
        overlap_mask = fixed_mask_np & nonlinear_mask_np
        nonlinear_nmi = normalized_mutual_information(
            fixed_array,
            nonlinear_warped_np,
            overlap_mask,
        )
        nonlinear_mask_dice = dice_binary(
            fixed_mask_np,
            nonlinear_mask_np,
        )
        nonlinear_center_distance_mm = float(
            np.linalg.norm(fixed_center - nonlinear_center)
        )

        raw_atlas_support = syn_full_fine_raw > 0
        raw_support_count = int(raw_atlas_support.sum())
        raw_inside_count = int(
            np.logical_and(raw_atlas_support, fixed_mask_np).sum()
        )
        raw_inside_fraction = (
            float(raw_inside_count / raw_support_count)
            if raw_support_count
            else 0.0
        )
        full_brain_coverage = float(
            np.logical_and(raw_atlas_support, fixed_mask_np).sum()
            / max(fixed_mask_np.sum(), 1)
        )
        distance_inside_brain = ndimage.distance_transform_edt(
            fixed_mask_np,
            sampling=spacing,
        )
        brain_core = distance_inside_brain >= 5.0
        core_coverage = float(
            np.logical_and(raw_atlas_support, brain_core).sum()
            / max(brain_core.sum(), 1)
        )

        present_vascular_labels = sorted(
            int(value)
            for value in np.unique(syn_vascular_fine)
            if 1 <= int(value) <= 30
        )
        (
            present_homolog_pairs,
            correctly_ordered_pairs,
            homolog_x_differences,
        ) = homolog_pair_metrics(syn_full_fine_clipped, patient_affine)

        jacobian_in_mask = jacobian_np[fixed_mask_np]
        jacobian_nonpositive_fraction = float(
            np.mean(jacobian_in_mask <= 0)
        )
        jacobian_p01 = float(np.quantile(jacobian_in_mask, 0.01))
        jacobian_median = float(np.median(jacobian_in_mask))
        jacobian_p99 = float(np.quantile(jacobian_in_mask, 0.99))

        safety_checks = {
            'fixed_orientation_is_ras': (
                ''.join(nib.aff2axcodes(patient_affine)) == 'RAS'
            ),
            'brain_mask_at_least_0p99_inside_crop': (
                brain_inside_crop_fraction >= 0.99
            ),
            'no_unexpected_labels': len(unexpected_labels) == 0,
            'at_least_24_of_30_vascular_labels_present': (
                len(present_vascular_labels) >= 24
            ),
            'at_least_12_homolog_pairs_present': (
                present_homolog_pairs >= 12
            ),
            'all_present_pairs_have_correct_left_right_order': (
                correctly_ordered_pairs == present_homolog_pairs
            ),
            'no_nonpositive_jacobians_in_fixed_mask': (
                jacobian_nonpositive_fraction <= 1e-6
            ),
            'jacobian_p01_at_least_0p10': jacobian_p01 >= 0.10,
            'jacobian_p99_at_most_5p0': jacobian_p99 <= 5.0,
            'raw_atlas_at_least_0p80_inside_brain': (
                raw_inside_fraction >= 0.80
            ),
            'full_brain_coverage_at_least_0p80': (
                full_brain_coverage >= 0.80
            ),
            'brain_core_coverage_at_least_0p85': core_coverage >= 0.85,
            'nonlinear_mask_dice_at_least_0p70': (
                nonlinear_mask_dice >= 0.70
            ),
            'nonlinear_center_distance_at_most_25mm': (
                nonlinear_center_distance_mm <= 25.0
            ),
            'nonlinear_nmi_is_finite': bool(np.isfinite(nonlinear_nmi)),
            'clipped_labels_remain_inside_brain': bool(
                not np.any((syn_full_fine_clipped > 0) & ~fixed_mask_np)
            ),
        }
        automated_safety_passed = bool(all(safety_checks.values()))

        output_sha256 = {
            path.name: file_sha256(path)
            for path in sorted(local_case_dir.iterdir())
            if path.is_file()
        }
        elapsed_seconds = float(time.time() - start_time)
        metrics = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'completed': True,
            'registration_version': REGISTRATION_VERSION,
            'bulk_code_revision': BULK_CODE_REVISION,
            'registration_signature': registration_signature,
            'subject_id': subject_id,
            'dev_split': dev_split,
            'npz_path': str(npz_path),
            'npz_sha256': observed_npz_sha256,
            'synthstrip_mask_path': str(mask_path),
            'synthstrip_mask_sha256': observed_mask_sha256,
            'cow_prior_valid': as_bool(manifest_row['cow_prior_valid']),
            'fixed_shape': list(fixed_array.shape),
            'fixed_mask_voxels': int(fixed_mask_np.sum()),
            'brain_inside_crop_fraction': brain_inside_crop_fraction,
            'nonlinear_mask_dice': nonlinear_mask_dice,
            'nonlinear_nmi': nonlinear_nmi,
            'nonlinear_center_distance_mm': nonlinear_center_distance_mm,
            'raw_atlas_inside_brain_fraction': raw_inside_fraction,
            'full_brain_coverage_fraction': full_brain_coverage,
            'brain_core_coverage_fraction': core_coverage,
            'present_vascular_label_count': len(present_vascular_labels),
            'present_vascular_labels': present_vascular_labels,
            'present_homolog_pairs': present_homolog_pairs,
            'correctly_ordered_homolog_pairs': correctly_ordered_pairs,
            'minimum_right_minus_left_centroid_mm': (
                float(min(homolog_x_differences))
                if homolog_x_differences
                else float('nan')
            ),
            'jacobian_min': float(np.min(jacobian_in_mask)),
            'jacobian_p01': jacobian_p01,
            'jacobian_median': jacobian_median,
            'jacobian_p99': jacobian_p99,
            'jacobian_max': float(np.max(jacobian_in_mask)),
            'jacobian_nonpositive_fraction': (
                jacobian_nonpositive_fraction
            ),
            'forward_transform_files': copied_forward,
            'inverse_transform_files': copied_inverse,
            'fine_labels_file': syn_fine_path.name,
            'vascular30_file': syn_vascular_path.name,
            'level2_file': syn_level2_path.name,
            'atlas_valid_mask_file': atlas_valid_path.name,
            'jacobian_file': jacobian_path.name,
            'visual_qa_file': qa_path.name,
            'elapsed_seconds': elapsed_seconds,
            'target_arrays_read': False,
            'automated_safety_checks': safety_checks,
            'automated_safety_passed': automated_safety_passed,
            'manual_visual_qa_approved': False,
            'output_sha256': output_sha256,
        }
        atomic_write_json(metrics, local_case_dir / 'complete.json')

        shutil.copytree(local_case_dir, subject_dir)
        committed = validate_completed_case(subject_dir, manifest_row)
        if committed is None:
            quarantine_incomplete_directory(subject_dir)
            raise RuntimeError(
                f'{subject_id} committed output failed resume validation.'
            )

        print(
            f'Completed {subject_id}: '
            f'safety={automated_safety_passed}, '
            f'labels={len(present_vascular_labels)}/30, '
            f'maskDice={nonlinear_mask_dice:.4f}, '
            f'core={core_coverage:.4f}, '
            f'Jac<=0={jacobian_nonpositive_fraction:.3g}, '
            f'{elapsed_seconds / 60:.1f} min'
        )
        return committed, False
    finally:
        if local_parent.exists():
            shutil.rmtree(local_parent)


print('Trial8 bulk SyNRA vascular-atlas registration')
print(f'Registration version: {REGISTRATION_VERSION}')
print(f'Frozen subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'ANTsPy version: {getattr(ants, "__version__", "unknown")}')
print(f'SciPy version: {scipy.__version__}')
print(f'ITK threads: {ITK_THREADS}')
print('Registration compute: CPU (ANTsPy)')
print('GPU may remain attached but is not used by ANTsPy registration.')
print('Target arrays read: False')
print('Study trainable parameters: 0 (atlas preprocessing only)')
print(f'Case-resumable output: {CASE_ROOT}')

records = []
failure_rows = []
reused_count = 0
computed_count = 0
run_start = time.time()

for case_index, row in enumerate(
    manifest_df.sort_values('subject_id').to_dict('records'),
    start=1,
):
    subject_id = str(row['subject_id'])
    print('\n' + '=' * 80)
    print(f'Bulk registration {case_index}/{EXPECTED_SUBJECTS}: {subject_id}')
    print('=' * 80)
    try:
        record, reused = process_subject(row)
        records.append(record)
        reused_count += int(reused)
        computed_count += int(not reused)
    except Exception as error:
        failure_rows.append({
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'subject_id': subject_id,
            'dev_split': str(row['dev_split']),
            'error_type': type(error).__name__,
            'error': str(error),
        })
        print(f'FAILED {subject_id}: {type(error).__name__}: {error}')

    flat_records = [
        {
            key: value
            for key, value in record.items()
            if not isinstance(value, (dict, list))
        }
        for record in records
    ]
    if flat_records:
        atomic_write_csv(
            pd.DataFrame(flat_records).sort_values('subject_id'),
            AUDIT_PATH,
        )
    if failure_rows:
        atomic_write_csv(pd.DataFrame(failure_rows), FAILURE_PATH)

if records:
    audit_df = pd.DataFrame([
        {
            key: value
            for key, value in record.items()
            if not isinstance(value, (dict, list))
        }
        for record in records
    ]).sort_values('subject_id').reset_index(drop=True)
else:
    audit_df = pd.DataFrame(columns=[
        'subject_id',
        'dev_split',
        'automated_safety_passed',
    ])
failure_df = pd.DataFrame(failure_rows)
if len(failure_df):
    atomic_write_csv(failure_df, FAILURE_PATH)
else:
    atomic_write_csv(
        pd.DataFrame(columns=[
            'created_at_utc',
            'subject_id',
            'dev_split',
            'error_type',
            'error',
        ]),
        FAILURE_PATH,
    )

all_subjects_completed = bool(
    len(audit_df) == EXPECTED_SUBJECTS
    and audit_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    and not len(failure_df)
)
all_automated_safety_passed = bool(
    all_subjects_completed
    and audit_df['automated_safety_passed'].map(as_bool).all()
)

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'registration_version': REGISTRATION_VERSION,
    'bulk_code_revision': BULK_CODE_REVISION,
    'registration_signature': registration_signature,
    'expected_subjects': EXPECTED_SUBJECTS,
    'completed_subjects': int(len(audit_df)),
    'computed_this_run': int(computed_count),
    'reused_this_run': int(reused_count),
    'failed_this_run': int(len(failure_df)),
    'all_subjects_completed': all_subjects_completed,
    'all_automated_safety_passed': all_automated_safety_passed,
    'manual_visual_qa_approved': False,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'registration_outputs_frozen': False,
    'model_training_authorized': False,
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'frozen_manifest_sha256': manifest_sha256,
    'freeze_gate_path': str(FREEZE_GATE_PATH),
    'freeze_gate_sha256': file_sha256(FREEZE_GATE_PATH),
    'config_path': str(CONFIG_PATH),
    'audit_path': str(AUDIT_PATH),
    'failure_path': str(FAILURE_PATH) if len(failure_df) else None,
    'case_root': str(CASE_ROOT),
    'elapsed_seconds_this_run': float(time.time() - run_start),
}
atomic_write_json(summary, SUMMARY_PATH)

print('\nBulk registration status by split:')
if len(audit_df):
    split_summary = audit_df.groupby('dev_split', as_index=False).agg(
        completed=('subject_id', 'count'),
        safety_passed=('automated_safety_passed', 'sum'),
        minimum_mask_dice=('nonlinear_mask_dice', 'min'),
        minimum_full_brain_coverage=(
            'full_brain_coverage_fraction',
            'min',
        ),
        minimum_core_coverage=('brain_core_coverage_fraction', 'min'),
        minimum_vascular_labels=('present_vascular_label_count', 'min'),
        maximum_center_distance_mm=(
            'nonlinear_center_distance_mm',
            'max',
        ),
        minimum_jacobian_p01=('jacobian_p01', 'min'),
        maximum_jacobian_p99=('jacobian_p99', 'max'),
    )
    display(split_summary)

if len(failure_df):
    print('\nRegistration failures:')
    display(failure_df)

print('\n' + '=' * 80)
print('TRIAL8 BULK SYNRA REGISTRATION RUN COMPLETE')
print('=' * 80)
print(f'Completed subjects:                 {len(audit_df)}/{EXPECTED_SUBJECTS}')
print(f'Computed this run:                  {computed_count}')
print(f'Reused this run:                    {reused_count}')
print(f'Failures this run:                  {len(failure_df)}')
print(f'All automated safety passed:        {all_automated_safety_passed}')
print('Manual visual QA approved:          False')
print('Target arrays read:                 False')
print('Study trainable parameters:         0 (atlas preprocessing only)')
print('Registration outputs frozen:        False')
print('Model training authorized:          False')
print(f'Case outputs:                       {CASE_ROOT}')
print(f'Audit:                              {AUDIT_PATH}')
print(f'Summary:                            {SUMMARY_PATH}')

if not all_subjects_completed:
    raise RuntimeError(
        'Bulk registration is incomplete. Rerun this same standalone cell to '
        'reuse valid completed cases and retry missing or failed cases.'
    )
if not all_automated_safety_passed:
    failed_subjects = audit_df.loc[
        ~audit_df['automated_safety_passed'].map(as_bool),
        'subject_id',
    ].astype(str).tolist()
    raise RuntimeError(
        'Bulk registration completed, but automated safety failed for: '
        + ', '.join(failed_subjects)
        + '. Do not freeze registration outputs or begin training.'
    )

print(
    '\nStop here and check the bulk metrics and selected visual QA images '
    'before freezing registration outputs.'
)


In [ ]:
import json
import math
import textwrap
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageOps
from google.colab import drive
from IPython.display import display


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
FROZEN_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
REGISTRATION_VERSION = 'v1_jhu_to_ncct_synra_synthstrip_r2_all148'
TRIAGE_VERSION = 'v1_exact_checks_and_visual_qa'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_frozen.csv'
)
REGISTRATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / FROZEN_PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'bulk_registration'
    / REGISTRATION_VERSION
)
CASE_ROOT = REGISTRATION_ROOT / 'cases'
BULK_AUDIT_PATH = REGISTRATION_ROOT / 'bulk_registration_audit.csv'
BULK_SUMMARY_PATH = REGISTRATION_ROOT / 'bulk_registration_summary.json'

OUTPUT_ROOT = REGISTRATION_ROOT / 'safety_triage' / TRIAGE_VERSION
FAILED_QA_ROOT = OUTPUT_ROOT / 'failed_case_montages'
PASS_QA_ROOT = OUTPUT_ROOT / 'passing_reference_montages'

EXACT_AUDIT_PATH = OUTPUT_ROOT / 'exact_safety_audit.csv'
FAILED_SUMMARY_PATH = OUTPUT_ROOT / 'failed_subject_summary.csv'
FAILED_MATRIX_PATH = OUTPUT_ROOT / 'failed_check_matrix.csv'
CHECK_FREQUENCY_PATH = OUTPUT_ROOT / 'failed_check_frequency.csv'
PASS_SAMPLE_PATH = OUTPUT_ROOT / 'passing_reference_sample.csv'
SPLIT_SUMMARY_PATH = OUTPUT_ROOT / 'split_safety_summary.csv'
SUMMARY_PATH = OUTPUT_ROOT / 'safety_triage_summary.json'
FREQUENCY_FIGURE_PATH = OUTPUT_ROOT / 'failed_check_frequency.png'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_REGISTRATION_VERSION = REGISTRATION_VERSION


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    if isinstance(value, str):
        normalized = value.strip().lower()
        if normalized in {'true', '1', 'yes'}:
            return True
        if normalized in {'false', '0', 'no'}:
            return False
    raise ValueError(f'Cannot interpret as bool: {value!r}')


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    temporary_path.write_text(
        json.dumps(payload, indent=2, allow_nan=False),
        encoding='utf-8',
    )
    temporary_path.replace(path)


def write_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def numeric_margin(record):
    margins = [
        (float(record['brain_inside_crop_fraction']) - 0.99) / 0.01,
        (float(record['present_vascular_label_count']) - 24.0) / 24.0,
        (float(record['present_homolog_pairs']) - 12.0) / 12.0,
        (float(record['jacobian_p01']) - 0.10) / 0.10,
        (5.0 - float(record['jacobian_p99'])) / 5.0,
        (float(record['raw_atlas_inside_brain_fraction']) - 0.80) / 0.80,
        (float(record['full_brain_coverage_fraction']) - 0.80) / 0.80,
        (float(record['brain_core_coverage_fraction']) - 0.85) / 0.85,
        (float(record['nonlinear_mask_dice']) - 0.70) / 0.70,
        (25.0 - float(record['nonlinear_center_distance_mm'])) / 25.0,
    ]
    return float(min(margins))


def priority_from_failures(failed_checks):
    critical_checks = {
        'fixed_orientation_is_ras',
        'no_unexpected_labels',
        'all_present_pairs_have_correct_left_right_order',
        'no_nonpositive_jacobians_in_fixed_mask',
        'clipped_labels_remain_inside_brain',
    }
    high_checks = {
        'at_least_24_of_30_vascular_labels_present',
        'at_least_12_homolog_pairs_present',
        'jacobian_p01_at_least_0p10',
        'jacobian_p99_at_most_5p0',
        'brain_core_coverage_at_least_0p85',
        'nonlinear_mask_dice_at_least_0p70',
        'nonlinear_center_distance_at_most_25mm',
    }
    failed_set = set(failed_checks)
    if failed_set & critical_checks:
        return 'critical'
    if failed_set & high_checks or len(failed_set) >= 2:
        return 'high'
    return 'standard'


def make_montages(rows, destination, prefix, rows_per_page=4):
    destination.mkdir(parents=True, exist_ok=True)
    page_paths = []
    if not rows:
        return page_paths

    tile_width = 1200
    image_height = 675
    title_height = 110
    tile_height = title_height + image_height
    columns = 2
    grid_rows = 2
    page_size = columns * grid_rows

    for page_index, start in enumerate(range(0, len(rows), page_size), 1):
        page_rows = rows[start:start + page_size]
        canvas = Image.new(
            'RGB',
            (tile_width * columns, tile_height * grid_rows),
            color='white',
        )
        draw = ImageDraw.Draw(canvas)

        for local_index, row in enumerate(page_rows):
            grid_y, grid_x = divmod(local_index, columns)
            x0 = grid_x * tile_width
            y0 = grid_y * tile_height

            failed_text = row.get('failed_checks', '')
            title = (
                f"{row['subject_id']} | {row['dev_split']} | "
                f"priority={row.get('review_priority', 'reference')}\n"
                f"{failed_text if failed_text else 'automated safety passed'}"
            )
            wrapped = '\n'.join(textwrap.wrap(title, width=105))
            draw.multiline_text(
                (x0 + 12, y0 + 10),
                wrapped,
                fill='black',
                spacing=4,
            )

            qa_path = Path(row['qa_path'])
            with Image.open(qa_path) as source_image:
                image = source_image.convert('RGB')
                contained = ImageOps.contain(
                    image,
                    (tile_width - 20, image_height - 20),
                )
                image_x = x0 + (tile_width - contained.width) // 2
                image_y = y0 + title_height + (
                    image_height - contained.height
                ) // 2
                canvas.paste(contained, (image_x, image_y))

        page_path = destination / f'{prefix}_page_{page_index:02d}.png'
        canvas.save(page_path, optimize=True)
        page_paths.append(page_path)

    return page_paths


required_paths = [
    FROZEN_MANIFEST_PATH,
    REGISTRATION_ROOT,
    CASE_ROOT,
    BULK_AUDIT_PATH,
    BULK_SUMMARY_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 safety triage prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
bulk_audit_df = pd.read_csv(BULK_AUDIT_PATH)
bulk_summary = json.loads(BULK_SUMMARY_PATH.read_text(encoding='utf-8'))

required_manifest_columns = {'subject_id', 'dev_split'}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        f'Frozen manifest is missing columns: {missing_manifest_columns}'
    )

manifest_df = manifest_df[['subject_id', 'dev_split']].copy()
manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
manifest_df['dev_split'] = manifest_df['dev_split'].astype(str)
manifest_subjects = set(manifest_df['subject_id'])

preflight_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_are_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
    'bulk_summary_all_subjects_completed': as_bool(
        bulk_summary.get('all_subjects_completed', False)
    ),
    'bulk_summary_completed_148': (
        int(bulk_summary.get('completed_subjects', -1)) == EXPECTED_SUBJECTS
    ),
    'bulk_summary_target_arrays_not_read': not as_bool(
        bulk_summary.get('target_arrays_read', True)
    ),
    'bulk_summary_outputs_not_frozen': not as_bool(
        bulk_summary.get('registration_outputs_frozen', True)
    ),
    'bulk_summary_training_not_authorized': not as_bool(
        bulk_summary.get('model_training_authorized', True)
    ),
}

if not all(preflight_checks.values()):
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'Trial8 safety triage preflight failed. Check: ' + ', '.join(failed)
    )

records = []
check_names = set()
integrity_errors = []

for position, manifest_row in manifest_df.sort_values('subject_id').iterrows():
    subject_id = manifest_row['subject_id']
    expected_split = manifest_row['dev_split']
    case_dir = CASE_ROOT / subject_id
    complete_path = case_dir / 'complete.json'

    if not complete_path.exists():
        integrity_errors.append(f'{subject_id}: missing complete.json')
        continue

    try:
        record = json.loads(complete_path.read_text(encoding='utf-8'))
    except Exception as exc:
        integrity_errors.append(f'{subject_id}: unreadable complete.json: {exc}')
        continue

    checks = record.get('automated_safety_checks')
    if not isinstance(checks, dict) or not checks:
        integrity_errors.append(f'{subject_id}: missing safety-check mapping')
        continue

    normalized_checks = {}
    try:
        normalized_checks = {
            str(name): as_bool(value) for name, value in checks.items()
        }
    except Exception as exc:
        integrity_errors.append(f'{subject_id}: invalid safety check: {exc}')
        continue

    check_names.update(normalized_checks)
    qa_path = case_dir / str(record.get('visual_qa_file', ''))
    failed_checks = sorted(
        name for name, passed in normalized_checks.items() if not passed
    )
    recomputed_safety = not failed_checks

    case_checks = {
        'subject_id_matches': str(record.get('subject_id')) == subject_id,
        'split_matches': str(record.get('dev_split')) == expected_split,
        'registration_version_matches': (
            str(record.get('registration_version'))
            == EXPECTED_REGISTRATION_VERSION
        ),
        'case_marked_complete': as_bool(record.get('completed', False)),
        'target_arrays_not_read': not as_bool(
            record.get('target_arrays_read', True)
        ),
        'saved_safety_matches_checks': (
            as_bool(record.get('automated_safety_passed', False))
            == recomputed_safety
        ),
        'visual_qa_exists': qa_path.is_file(),
    }
    failed_case_integrity = [
        name for name, passed in case_checks.items() if not passed
    ]
    if failed_case_integrity:
        integrity_errors.append(
            f'{subject_id}: ' + ', '.join(failed_case_integrity)
        )
        continue

    flat_record = {
        key: value
        for key, value in record.items()
        if not isinstance(value, (dict, list))
    }
    flat_record['qa_path'] = str(qa_path)
    flat_record['failed_check_count'] = len(failed_checks)
    flat_record['failed_checks'] = '; '.join(failed_checks)
    flat_record['review_priority'] = priority_from_failures(failed_checks)
    flat_record['safety_margin'] = numeric_margin(record)
    flat_record['_checks'] = normalized_checks
    records.append(flat_record)

    completed_count = len(records)
    if completed_count == 1 or completed_count % 20 == 0:
        print(f'Loaded safety record {completed_count}/{EXPECTED_SUBJECTS}: {subject_id}')

if integrity_errors:
    preview = '\n'.join(integrity_errors[:20])
    raise RuntimeError(
        'Trial8 safety triage found case-integrity errors. Do not continue.\n'
        + preview
    )

if len(records) != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f'Expected {EXPECTED_SUBJECTS} valid case records, found {len(records)}.'
    )

record_subjects = {str(record['subject_id']) for record in records}
if record_subjects != manifest_subjects:
    missing = sorted(manifest_subjects - record_subjects)
    extra = sorted(record_subjects - manifest_subjects)
    raise RuntimeError(
        f'Case/manifest subject mismatch. Missing={missing}, extra={extra}'
    )

check_names = sorted(check_names)
expected_check_set = set(records[0]['_checks'])
inconsistent_check_sets = [
    record['subject_id']
    for record in records
    if set(record['_checks']) != expected_check_set
]
if inconsistent_check_sets:
    raise RuntimeError(
        'Safety-check schema differs across cases: '
        + ', '.join(inconsistent_check_sets)
    )

exact_rows = []
matrix_rows = []
for record in records:
    check_values = record.pop('_checks')
    exact_row = dict(record)
    exact_row.update({f'check__{name}': value for name, value in check_values.items()})
    exact_rows.append(exact_row)

    matrix_row = {
        'subject_id': record['subject_id'],
        'dev_split': record['dev_split'],
        'automated_safety_passed': record['automated_safety_passed'],
        'failed_check_count': record['failed_check_count'],
        'review_priority': record['review_priority'],
    }
    matrix_row.update({name: value for name, value in check_values.items()})
    matrix_rows.append(matrix_row)

exact_df = pd.DataFrame(exact_rows).sort_values('subject_id').reset_index(drop=True)
matrix_df = pd.DataFrame(matrix_rows).sort_values('subject_id').reset_index(drop=True)
exact_df['automated_safety_passed'] = exact_df[
    'automated_safety_passed'
].map(as_bool)
matrix_df['automated_safety_passed'] = matrix_df[
    'automated_safety_passed'
].map(as_bool)

bulk_subjects = set(bulk_audit_df['subject_id'].astype(str))
if bulk_subjects != manifest_subjects:
    raise RuntimeError('Bulk audit subject set does not match the frozen manifest.')

bulk_safety = {
    str(row.subject_id): as_bool(row.automated_safety_passed)
    for row in bulk_audit_df[['subject_id', 'automated_safety_passed']].itertuples(
        index=False
    )
}
triage_safety = {
    str(row.subject_id): as_bool(row.automated_safety_passed)
    for row in exact_df[['subject_id', 'automated_safety_passed']].itertuples(
        index=False
    )
}
if bulk_safety != triage_safety:
    raise RuntimeError('Recomputed safety flags do not match the bulk audit.')

failed_df = exact_df.loc[~exact_df['automated_safety_passed']].copy()
passed_df = exact_df.loc[exact_df['automated_safety_passed']].copy()

priority_order = {'critical': 0, 'high': 1, 'standard': 2}
failed_df['_priority_order'] = failed_df['review_priority'].map(priority_order)
failed_df = failed_df.sort_values(
    ['_priority_order', 'failed_check_count', 'safety_margin', 'subject_id'],
    ascending=[True, False, True, True],
).drop(columns=['_priority_order']).reset_index(drop=True)

failed_matrix_df = matrix_df.loc[
    ~matrix_df['automated_safety_passed']
].copy()
failed_matrix_df = failed_matrix_df.set_index('subject_id').loc[
    failed_df['subject_id']
].reset_index()

frequency_rows = []
for check_name in check_names:
    failure_count = int((~matrix_df[check_name].map(as_bool)).sum())
    frequency_rows.append({
        'check_name': check_name,
        'failed_subjects': failure_count,
        'failure_fraction': failure_count / EXPECTED_SUBJECTS,
    })
frequency_df = pd.DataFrame(frequency_rows).sort_values(
    ['failed_subjects', 'check_name'], ascending=[False, True]
).reset_index(drop=True)

split_summary_df = exact_df.groupby('dev_split', as_index=False).agg(
    subjects=('subject_id', 'count'),
    safety_passed=('automated_safety_passed', 'sum'),
    minimum_mask_dice=('nonlinear_mask_dice', 'min'),
    minimum_full_brain_coverage=('full_brain_coverage_fraction', 'min'),
    minimum_core_coverage=('brain_core_coverage_fraction', 'min'),
    minimum_vascular_labels=('present_vascular_label_count', 'min'),
    maximum_center_distance_mm=('nonlinear_center_distance_mm', 'max'),
    minimum_jacobian_p01=('jacobian_p01', 'min'),
    maximum_jacobian_p99=('jacobian_p99', 'max'),
)
split_summary_df['safety_failed'] = (
    split_summary_df['subjects'] - split_summary_df['safety_passed']
)
split_summary_df['pass_fraction'] = (
    split_summary_df['safety_passed'] / split_summary_df['subjects']
)

pass_sample_rows = []
for split_name in ['train', 'val', 'test']:
    split_passed = passed_df.loc[
        passed_df['dev_split'] == split_name
    ].sort_values(['safety_margin', 'subject_id']).reset_index(drop=True)
    if split_passed.empty:
        continue
    candidate_indices = sorted(set([
        0,
        len(split_passed) // 2,
        len(split_passed) - 1,
    ]))
    labels = ['borderline_pass', 'median_pass', 'strong_pass']
    for label, row_index in zip(labels, candidate_indices):
        row = split_passed.iloc[row_index].to_dict()
        row['reference_selection'] = label
        row['failed_checks'] = ''
        row['review_priority'] = 'reference'
        pass_sample_rows.append(row)

pass_sample_df = pd.DataFrame(pass_sample_rows)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
write_csv(exact_df, EXACT_AUDIT_PATH)
write_csv(failed_df, FAILED_SUMMARY_PATH)
write_csv(failed_matrix_df, FAILED_MATRIX_PATH)
write_csv(frequency_df, CHECK_FREQUENCY_PATH)
write_csv(pass_sample_df, PASS_SAMPLE_PATH)
write_csv(split_summary_df, SPLIT_SUMMARY_PATH)

nonzero_frequency_df = frequency_df.loc[
    frequency_df['failed_subjects'] > 0
].copy()
if len(nonzero_frequency_df):
    figure_height = max(4.5, 0.48 * len(nonzero_frequency_df) + 1.8)
    fig, ax = plt.subplots(figsize=(12, figure_height))
    plot_df = nonzero_frequency_df.sort_values('failed_subjects')
    ax.barh(plot_df['check_name'], plot_df['failed_subjects'], color='#b33a3a')
    for y_index, count in enumerate(plot_df['failed_subjects']):
        ax.text(count + 0.25, y_index, str(int(count)), va='center')
    ax.set_xlabel('Failed subjects')
    ax.set_title('Trial8 bulk registration safety-check failures')
    ax.set_xlim(0, max(plot_df['failed_subjects']) * 1.15 + 1)
    ax.grid(axis='x', alpha=0.25)
    fig.tight_layout()
    fig.savefig(FREQUENCY_FIGURE_PATH, dpi=180, bbox_inches='tight')
    plt.close(fig)

failed_montage_rows = failed_df[
    ['subject_id', 'dev_split', 'review_priority', 'failed_checks', 'qa_path']
].to_dict('records')
failed_montage_paths = make_montages(
    failed_montage_rows,
    FAILED_QA_ROOT,
    'failed_registration_qa',
)

pass_montage_rows = pass_sample_df[
    ['subject_id', 'dev_split', 'review_priority', 'failed_checks', 'qa_path']
].to_dict('records')
pass_montage_paths = make_montages(
    pass_montage_rows,
    PASS_QA_ROOT,
    'passing_reference_qa',
)

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'triage_version': TRIAGE_VERSION,
    'registration_version': REGISTRATION_VERSION,
    'subjects': int(len(exact_df)),
    'automated_safety_passed': int(exact_df['automated_safety_passed'].sum()),
    'automated_safety_failed': int((~exact_df['automated_safety_passed']).sum()),
    'critical_review_subjects': int(
        (failed_df['review_priority'] == 'critical').sum()
    ),
    'high_review_subjects': int(
        (failed_df['review_priority'] == 'high').sum()
    ),
    'standard_review_subjects': int(
        (failed_df['review_priority'] == 'standard').sum()
    ),
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'registration_outputs_modified': False,
    'registration_outputs_frozen': False,
    'model_training_authorized': False,
    'exact_audit_path': str(EXACT_AUDIT_PATH),
    'failed_subject_summary_path': str(FAILED_SUMMARY_PATH),
    'failed_check_matrix_path': str(FAILED_MATRIX_PATH),
    'failed_check_frequency_path': str(CHECK_FREQUENCY_PATH),
    'passing_reference_sample_path': str(PASS_SAMPLE_PATH),
    'split_summary_path': str(SPLIT_SUMMARY_PATH),
    'failed_montage_paths': [str(path) for path in failed_montage_paths],
    'passing_montage_paths': [str(path) for path in pass_montage_paths],
}
write_json(SUMMARY_PATH, summary)

print('\nTrial8 exact failed-check frequency:')
display(nonzero_frequency_df)

print('\nTrial8 failed subjects ranked for visual review:')
display(failed_df[[
    'subject_id',
    'dev_split',
    'review_priority',
    'failed_check_count',
    'failed_checks',
    'nonlinear_mask_dice',
    'full_brain_coverage_fraction',
    'brain_core_coverage_fraction',
    'nonlinear_center_distance_mm',
    'present_vascular_label_count',
    'present_homolog_pairs',
    'correctly_ordered_homolog_pairs',
    'jacobian_p01',
    'jacobian_p99',
]])

print('\nTrial8 safety status by split:')
display(split_summary_df)

if FREQUENCY_FIGURE_PATH.exists():
    display(Image.open(FREQUENCY_FIGURE_PATH))

print('\nFailed-case visual QA montages:')
for montage_path in failed_montage_paths:
    print(montage_path)
    display(Image.open(montage_path))

print('\nPassing-reference visual QA montages:')
for montage_path in pass_montage_paths:
    print(montage_path)
    display(Image.open(montage_path))

print('\n' + '=' * 80)
print('TRIAL8 BULK REGISTRATION SAFETY TRIAGE COMPLETE')
print('=' * 80)
print(f'Subjects checked:                    {len(exact_df)}/148')
print(
    'Automated safety passed:           '
    f'{int(exact_df["automated_safety_passed"].sum())}'
)
print(f'Automated safety failed:           {len(failed_df)}')
print(
    'Critical / high / standard review: '
    f'{summary["critical_review_subjects"]} / '
    f'{summary["high_review_subjects"]} / '
    f'{summary["standard_review_subjects"]}'
)
print('Target arrays read:                 False')
print('Study trainable parameters:         0 (registration QA only)')
print('Registration outputs modified:      False')
print('Registration outputs frozen:        False')
print('Model training authorized:          False')
print(f'Exact safety audit:                 {EXACT_AUDIT_PATH}')
print(f'Failed-subject summary:             {FAILED_SUMMARY_PATH}')
print(f'Failed-check matrix:                {FAILED_MATRIX_PATH}')
print(f'Failed-check frequency:             {CHECK_FREQUENCY_PATH}')
print(f'QA output directory:                {OUTPUT_ROOT}')
print(f'Triage summary:                     {SUMMARY_PATH}')
print('\nNext: check every failed-case montage before generating affine fallbacks.')


In [ ]:
import os
import sys
import json
import time
import signal
import subprocess
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
ENVIRONMENT_RECORD_PATH = (
    PROJECT_ROOT / 'configs' / 'trial8_registration_environment.json'
)

PINNED_ANTSPYX = '0.6.3'
PINNED_SCIPY = '1.16.3'
NUMPY_SPEC = 'numpy<2.4'


def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None


def run_clean_import_smoke_test():
    smoke_code = r'''
import json
import numpy as np
import scipy
from scipy import ndimage
import ants

fixed = ants.from_numpy(np.zeros((8, 8, 8), dtype=np.float32))
moving = ants.from_numpy(np.zeros((8, 8, 8), dtype=np.float32))

print(json.dumps({
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "antspyx": getattr(ants, "__version__", "unknown"),
    "ants_dimension": int(fixed.dimension),
    "ndimage_available": callable(ndimage.binary_fill_holes),
}))
'''
    result = subprocess.run(
        [sys.executable, '-c', smoke_code],
        text=True,
        capture_output=True,
    )

    if result.returncode != 0:
        return False, None, result.stderr.strip()

    output_lines = [line for line in result.stdout.splitlines() if line.strip()]
    if not output_lines:
        return False, None, 'Clean import smoke test returned no output.'

    try:
        versions = json.loads(output_lines[-1])
    except json.JSONDecodeError as exc:
        return False, None, f'Could not parse smoke-test output: {exc}'

    version_ok = (
        versions.get('scipy') == PINNED_SCIPY
        and versions.get('antspyx') == PINNED_ANTSPYX
        and versions.get('ndimage_available') is True
        and versions.get('ants_dimension') == 3
    )

    if not version_ok:
        return False, versions, 'Installed versions do not match the Trial8 pins.'

    return True, versions, None


def install_registration_stack():
    dependency_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--upgrade',
        '--no-cache-dir',
        NUMPY_SPEC,
        'pandas',
        'nibabel',
        'pyyaml',
        'statsmodels',
        'webcolors',
        'matplotlib',
        'Pillow',
        'requests',
        'scikit-learn',
        'packaging',
    ]
    subprocess.check_call(dependency_command)

    
    
    pinned_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--force-reinstall',
        '--no-deps',
        '--no-cache-dir',
        f'scipy=={PINNED_SCIPY}',
        f'antspyx=={PINNED_ANTSPYX}',
    ]
    subprocess.check_call(pinned_command)


metadata_versions = {
    'antspyx': installed_version('antspyx'),
    'scipy': installed_version('scipy'),
    'numpy': installed_version('numpy'),
}

loaded_sensitive_roots = sorted({
    module_name.split('.')[0]
    for module_name in sys.modules
    if module_name.split('.')[0] in {'scipy', 'ants', 'statsmodels'}
})

smoke_passed, smoke_versions, smoke_error = run_clean_import_smoke_test()
installation_performed = False

if not smoke_passed:
    print('Trial8 registration environment needs repair.')
    print(f'Installed metadata before repair: {metadata_versions}')
    if smoke_versions is not None:
        print(f'Clean-process versions before repair: {smoke_versions}')
    if smoke_error:
        print(f'Clean-process error: {smoke_error}')

    install_registration_stack()
    installation_performed = True

    smoke_passed, smoke_versions, smoke_error = run_clean_import_smoke_test()
    if not smoke_passed:
        raise RuntimeError(
            'Trial8 registration environment repair failed. '
            f'Clean-process error: {smoke_error}'
        )

ENVIRONMENT_RECORD_PATH.parent.mkdir(parents=True, exist_ok=True)
environment_record = {
    'checked_at_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': sys.version,
    'python_executable': sys.executable,
    'pinned_antspyx': PINNED_ANTSPYX,
    'pinned_scipy': PINNED_SCIPY,
    'numpy_specification': NUMPY_SPEC,
    'clean_process_versions': smoke_versions,
    'clean_import_smoke_test_passed': bool(smoke_passed),
    'installation_performed': bool(installation_performed),
}
ENVIRONMENT_RECORD_PATH.write_text(
    json.dumps(environment_record, indent=2),
    encoding='utf-8',
)

restart_required = installation_performed or bool(loaded_sensitive_roots)

if restart_required:
    print('\nTrial8 ANTs/SciPy files are now consistent on disk.')
    if loaded_sensitive_roots:
        print(f'Loaded modules requiring a clean restart: {loaded_sensitive_roots}')
    print(f'Verified clean-process versions: {smoke_versions}')
    print(f'Saved environment record: {ENVIRONMENT_RECORD_PATH}')
    print('Colab will restart now to clear stale imported modules.')
    print('After reconnecting, run this environment cell once more.')
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), signal.SIGKILL)

print('\n' + '=' * 80)
print('TRIAL8 REGISTRATION ENVIRONMENT READY')
print('=' * 80)
print(f'Python:   {sys.version.split()[0]}')
print(f'NumPy:    {smoke_versions["numpy"]}')
print(f'SciPy:    {smoke_versions["scipy"]}')
print(f'ANTsPyX:  {smoke_versions["antspyx"]}')
print(f'Record:   {ENVIRONMENT_RECORD_PATH}')
print('\nNext: run the standalone Trial8 registration preflight cell.')


In [ ]:
import os
import sys
import json
import time
import shutil
import hashlib
import tempfile
import importlib.metadata as metadata
from datetime import datetime, timezone
from pathlib import Path


ITK_THREADS = min(4, os.cpu_count() or 2)
os.environ['ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS'] = str(ITK_THREADS)

PINNED_ANTSPYX = '0.6.3'
PINNED_SCIPY = '1.16.3'
CODE_REVISION = 'r1_affine_fallback_nine_case_candidate'


def installed_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None


dependency_errors = []
if installed_version('antspyx') != PINNED_ANTSPYX:
    dependency_errors.append(
        f'antspyx must be {PINNED_ANTSPYX}; '
        f'found {installed_version("antspyx")}'
    )
if installed_version('scipy') != PINNED_SCIPY:
    dependency_errors.append(
        f'scipy must be {PINNED_SCIPY}; '
        f'found {installed_version("scipy")}'
    )
if dependency_errors:
    raise RuntimeError(
        'Trial8 registration environment is not ready. Run the standalone '
        'trial8_registration_environment_setup.py cell, allow its requested '
        'runtime restart, run that setup cell once more, and then rerun this '
        'fallback cell. Problems: ' + '; '.join(dependency_errors)
    )

try:
    import numpy as np
    import pandas as pd
    import nibabel as nib
    import scipy
    from scipy import ndimage
    import matplotlib.pyplot as plt
    import ants
except Exception as exc:
    raise RuntimeError(
        'The pinned Trial8 registration stack could not be imported. Run the '
        'standalone registration-environment setup cell before this block.'
    ) from exc

from google.colab import drive
from IPython.display import display, Image as IPythonImage


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
PARENT_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
FROZEN_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
SOURCE_REGISTRATION_VERSION = 'v1_jhu_to_ncct_synra_synthstrip_r2_all148'
TRIAGE_VERSION = 'v1_exact_checks_and_visual_qa'
FALLBACK_VERSION = 'v1_syn_affine_or_disabled_candidate'
SCHEMA_VERSION = 'vascular_hierarchy_v1'

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_frozen.csv'
)
SOURCE_REGISTRATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / FROZEN_PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'bulk_registration'
    / SOURCE_REGISTRATION_VERSION
)
SOURCE_CASE_ROOT = SOURCE_REGISTRATION_ROOT / 'cases'
SOURCE_SUMMARY_PATH = SOURCE_REGISTRATION_ROOT / 'bulk_registration_summary.json'
TRIAGE_ROOT = SOURCE_REGISTRATION_ROOT / 'safety_triage' / TRIAGE_VERSION
EXACT_TRIAGE_PATH = TRIAGE_ROOT / 'exact_safety_audit.csv'
TRIAGE_SUMMARY_PATH = TRIAGE_ROOT / 'safety_triage_summary.json'

ATLAS_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / PARENT_PREPROCESSING_VERSION
    / 'vascular_atlas'
)
JHU_REFERENCE_PATH = (
    ATLAS_ROOT
    / 'reference'
    / 'JHU_MNI_SS'
    / 'JHU_MNI_SS_T1_brain_182x218x182.nii.gz'
)
FULL_FINE_ATLAS_PATH = (
    ATLAS_ROOT
    / 'source'
    / 'Arterial_Atlas'
    / 'data'
    / 'Atlas_182'
    / 'ArterialAtlas.nii'
)
SCHEMA_CSV_PATH = (
    ATLAS_ROOT
    / 'schema'
    / SCHEMA_VERSION
    / 'vascular_label_schema.csv'
)

OUTPUT_ROOT = SOURCE_REGISTRATION_ROOT / 'fallback_candidates' / FALLBACK_VERSION
CASE_ROOT = OUTPUT_ROOT / 'cases'
QUARANTINE_ROOT = OUTPUT_ROOT / 'incomplete_quarantine'
COMPARISON_AUDIT_PATH = OUTPUT_ROOT / 'affine_candidate_comparison.csv'
SELECTION_MANIFEST_PATH = OUTPUT_ROOT / 'candidate_selection_manifest.csv'
SUMMARY_PATH = OUTPUT_ROOT / 'fallback_candidate_summary.json'
CONFIG_PATH = OUTPUT_ROOT / 'fallback_candidate_config.json'
MONTAGE_ROOT = OUTPUT_ROOT / 'comparison_montages'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SPACING = np.asarray([0.8, 0.8, 2.0], dtype=np.float64)

DEFINITE_FALLBACK_SUBJECTS = [
    'sub-stroke0015',
    'sub-stroke0079',
    'sub-stroke0089',
    'sub-stroke0094',
    'sub-stroke0141',
    'sub-stroke0149',
    'sub-stroke0157',
]
BORDERLINE_FALLBACK_SUBJECTS = [
    'sub-stroke0082',
    'sub-stroke0162',
]
FALLBACK_SUBJECTS = sorted(
    DEFINITE_FALLBACK_SUBJECTS + BORDERLINE_FALLBACK_SUBJECTS
)




ADVISORY_CHECKS = {'raw_atlas_at_least_0p80_inside_brain'}


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes'}:
        return True
    if normalized in {'false', '0', 'no'}:
        return False
    raise ValueError(f'Cannot interpret as bool: {value!r}')


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def object_sha256(payload):
    encoded = json.dumps(
        payload,
        sort_keys=True,
        separators=(',', ':'),
    ).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def atomic_write_json(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + '.tmp')
    temporary_path.write_text(
        json.dumps(payload, indent=2, allow_nan=False),
        encoding='utf-8',
    )
    temporary_path.replace(path)


def atomic_write_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def assert_project_path_is_safe(path):
    resolved_path = Path(path).resolve()
    resolved_root = PROJECT_ROOT.resolve()
    if resolved_path != resolved_root and resolved_root not in resolved_path.parents:
        raise ValueError(f'Unsafe path outside project root: {resolved_path}')


def load_effective_3d(path, dtype):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 3:
        raise RuntimeError(f'{path} is not effectively 3D: {array.shape}')
    return image, np.asarray(array, dtype=dtype)


def save_nifti(data, affine, path, dtype):
    array = np.asarray(data, dtype=dtype)
    image = nib.Nifti1Image(array, np.asarray(affine, dtype=np.float64))
    image.set_sform(np.asarray(affine, dtype=np.float64), code=2)
    image.set_qform(np.asarray(affine, dtype=np.float64), code=1)
    nib.save(image, str(path))


def dice_binary(first, second, eps=1e-6):
    first = np.asarray(first, dtype=bool)
    second = np.asarray(second, dtype=bool)
    return float(
        (2.0 * np.logical_and(first, second).sum() + eps)
        / (first.sum() + second.sum() + eps)
    )


def normalized_mutual_information(first, second, mask, bins=64):
    first = np.asarray(first, dtype=np.float32)
    second = np.asarray(second, dtype=np.float32)
    valid_mask = (
        np.asarray(mask, dtype=bool)
        & np.isfinite(first)
        & np.isfinite(second)
    )
    if valid_mask.sum() < 100:
        return float('nan')
    x = first[valid_mask]
    y = second[valid_mask]
    histogram, _, _ = np.histogram2d(x, y, bins=bins)
    probability = histogram / max(histogram.sum(), 1.0)
    px = probability.sum(axis=1)
    py = probability.sum(axis=0)
    nonzero_joint = probability > 0
    nonzero_x = px > 0
    nonzero_y = py > 0
    h_joint = -float(
        np.sum(probability[nonzero_joint] * np.log(probability[nonzero_joint]))
    )
    h_x = -float(np.sum(px[nonzero_x] * np.log(px[nonzero_x])))
    h_y = -float(np.sum(py[nonzero_y] * np.log(py[nonzero_y])))
    return float((h_x + h_y) / h_joint) if h_joint > 0 else float('nan')


def center_of_mass_world(mask, affine):
    coordinates = np.argwhere(np.asarray(mask, dtype=bool))
    if not len(coordinates):
        return np.full(3, np.nan, dtype=np.float64)
    center_voxel = coordinates.mean(axis=0)
    return np.asarray(
        nib.affines.apply_affine(affine, center_voxel),
        dtype=np.float64,
    )


def homolog_pair_metrics(labels, affine):
    present_pairs = 0
    correctly_ordered_pairs = 0
    x_differences = []
    for left_label in range(1, 30, 2):
        right_label = left_label + 1
        left_mask = labels == left_label
        right_mask = labels == right_label
        if left_mask.any() and right_mask.any():
            present_pairs += 1
            left_x = center_of_mass_world(left_mask, affine)[0]
            right_x = center_of_mass_world(right_mask, affine)[0]
            difference = float(right_x - left_x)
            x_differences.append(difference)
            if difference > 0:
                correctly_ordered_pairs += 1
    return present_pairs, correctly_ordered_pairs, x_differences


def label_boundary(labels):
    labels = np.asarray(labels)
    boundary = np.zeros(labels.shape, dtype=bool)
    for axis in range(3):
        lower = [slice(None)] * 3
        upper = [slice(None)] * 3
        lower[axis] = slice(0, -1)
        upper[axis] = slice(1, None)
        difference = labels[tuple(lower)] != labels[tuple(upper)]
        boundary[tuple(lower)] |= difference
        boundary[tuple(upper)] |= difference
    return boundary & (labels > 0)


def display_slice(array, axis, index):
    if axis == 0:
        image = array[index, :, :]
    elif axis == 1:
        image = array[:, index, :]
    else:
        image = array[:, :, index]
    return np.rot90(image)


def final_use_checks_from_syn(record):
    checks = {
        str(name): as_bool(value)
        for name, value in record['automated_safety_checks'].items()
    }
    return {
        name: value
        for name, value in checks.items()
        if name not in ADVISORY_CHECKS
    }


def quality_confidence(metrics):
    components = [
        np.clip((float(metrics['nonlinear_mask_dice']) - 0.70) / 0.25, 0, 1),
        np.clip(
            (float(metrics['full_brain_coverage_fraction']) - 0.80) / 0.18,
            0,
            1,
        ),
        np.clip(
            (float(metrics['brain_core_coverage_fraction']) - 0.85) / 0.14,
            0,
            1,
        ),
        np.clip(
            (25.0 - float(metrics['nonlinear_center_distance_mm'])) / 25.0,
            0,
            1,
        ),
        np.clip(float(metrics['present_vascular_label_count']) / 30.0, 0, 1),
    ]
    return float(np.mean(components))


def save_comparison_qa(
    fixed_array,
    fixed_mask,
    syn_warped,
    affine_warped,
    syn_labels,
    affine_labels,
    cow_label,
    lvo_proximity,
    output_path,
    subject_id,
    syn_metrics,
    affine_metrics,
    recommendation,
):
    center = np.rint(np.argwhere(fixed_mask).mean(axis=0)).astype(int)
    axes = [2, 1, 0]
    axis_names = ['axial', 'coronal', 'sagittal']
    indices = [int(center[axis]) for axis in axes]
    mask_boundary = fixed_mask & ~ndimage.binary_erosion(fixed_mask)
    syn_boundary = label_boundary(syn_labels)
    affine_boundary = label_boundary(affine_labels)
    syn_ventricles = np.isin(syn_labels, [31, 32])
    affine_ventricles = np.isin(affine_labels, [31, 32])
    vessel_mask = cow_label > 0
    lvo_mask = lvo_proximity >= 0.5

    figure, plot_axes = plt.subplots(3, 5, figsize=(23, 13))
    for row, (axis, axis_name, index) in enumerate(
        zip(axes, axis_names, indices)
    ):
        fixed_slice = display_slice(fixed_array, axis, index)
        mask_slice = display_slice(mask_boundary, axis, index)
        syn_warped_slice = display_slice(syn_warped, axis, index)
        affine_warped_slice = display_slice(affine_warped, axis, index)
        syn_labels_slice = display_slice(syn_labels, axis, index)
        affine_labels_slice = display_slice(affine_labels, axis, index)
        syn_boundary_slice = display_slice(syn_boundary, axis, index)
        affine_boundary_slice = display_slice(affine_boundary, axis, index)
        syn_ventricle_slice = display_slice(syn_ventricles, axis, index)
        affine_ventricle_slice = display_slice(affine_ventricles, axis, index)
        vessel_slice = display_slice(vessel_mask, axis, index)
        lvo_slice = display_slice(lvo_mask, axis, index)

        plot_axes[row, 0].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 0].imshow(
            np.ma.masked_where(~mask_slice, mask_slice),
            cmap='autumn',
            alpha=0.95,
        )
        plot_axes[row, 0].set_title(f'{axis_name}: NCCT + mask')

        plot_axes[row, 1].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 1].imshow(syn_warped_slice, cmap='magma', alpha=0.40)
        plot_axes[row, 1].set_title('Existing SyN JHU')

        plot_axes[row, 2].imshow(fixed_slice, cmap='gray', vmin=0, vmax=1)
        plot_axes[row, 2].imshow(
            affine_warped_slice,
            cmap='magma',
            alpha=0.40,
        )
        plot_axes[row, 2].set_title('Affine-only JHU')

        for column, labels_slice, boundary_slice, ventricle_slice, title in [
            (3, syn_labels_slice, syn_boundary_slice, syn_ventricle_slice, 'SyN territories'),
            (4, affine_labels_slice, affine_boundary_slice, affine_ventricle_slice, 'Affine territories'),
        ]:
            plot_axes[row, column].imshow(
                fixed_slice,
                cmap='gray',
                vmin=0,
                vmax=1,
            )
            plot_axes[row, column].imshow(
                np.ma.masked_where(labels_slice == 0, labels_slice),
                cmap='turbo',
                alpha=0.28,
                vmin=1,
                vmax=32,
            )
            plot_axes[row, column].imshow(
                np.ma.masked_where(~boundary_slice, boundary_slice),
                cmap='cool',
                alpha=0.85,
            )
            if ventricle_slice.any():
                plot_axes[row, column].contour(
                    ventricle_slice,
                    levels=[0.5],
                    colors='lime',
                    linewidths=0.8,
                )
            if vessel_slice.any():
                plot_axes[row, column].contour(
                    vessel_slice,
                    levels=[0.5],
                    colors='magenta',
                    linewidths=0.7,
                )
            if lvo_slice.any():
                plot_axes[row, column].contour(
                    lvo_slice,
                    levels=[0.5],
                    colors='yellow',
                    linewidths=1.0,
                )
            plot_axes[row, column].set_title(title)

        for column in range(5):
            plot_axes[row, column].axis('off')

    figure.suptitle(
        f'{subject_id}: SyN versus affine-only fallback\n'
        f'SyN Dice/core/full={syn_metrics["nonlinear_mask_dice"]:.3f}/'
        f'{syn_metrics["brain_core_coverage_fraction"]:.3f}/'
        f'{syn_metrics["full_brain_coverage_fraction"]:.3f} | '
        f'Affine={affine_metrics["nonlinear_mask_dice"]:.3f}/'
        f'{affine_metrics["brain_core_coverage_fraction"]:.3f}/'
        f'{affine_metrics["full_brain_coverage_fraction"]:.3f} | '
        f'proposal={recommendation}\n'
        'orange=brain, cyan=territories, lime=ventricles, '
        'magenta=CoW, yellow=LVO',
        fontsize=13,
    )
    figure.tight_layout(rect=[0, 0, 1, 0.93])
    figure.savefig(output_path, dpi=125, bbox_inches='tight')
    plt.close(figure)


def make_montages(case_rows, destination):
    destination.mkdir(parents=True, exist_ok=True)
    page_paths = []
    for page_index, start in enumerate(range(0, len(case_rows), 3), 1):
        page_rows = case_rows[start:start + 3]
        figure, axes = plt.subplots(
            len(page_rows),
            1,
            figsize=(20, 9 * len(page_rows)),
        )
        if len(page_rows) == 1:
            axes = [axes]
        for axis, row in zip(axes, page_rows):
            image = plt.imread(row['comparison_qa_path'])
            axis.imshow(image)
            axis.set_title(
                f"{row['subject_id']} | proposal={row['proposed_mode']} | "
                f"affine safety={row['affine_final_use_safety_passed']}",
                fontsize=12,
            )
            axis.axis('off')
        figure.tight_layout()
        page_path = destination / f'affine_fallback_comparison_page_{page_index:02d}.png'
        figure.savefig(page_path, dpi=130, bbox_inches='tight')
        plt.close(figure)
        page_paths.append(page_path)
    return page_paths


required_paths = [
    FROZEN_MANIFEST_PATH,
    SOURCE_REGISTRATION_ROOT,
    SOURCE_CASE_ROOT,
    SOURCE_SUMMARY_PATH,
    EXACT_TRIAGE_PATH,
    TRIAGE_SUMMARY_PATH,
    JHU_REFERENCE_PATH,
    FULL_FINE_ATLAS_PATH,
    SCHEMA_CSV_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 fallback prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

for path in [OUTPUT_ROOT, CASE_ROOT, QUARANTINE_ROOT, MONTAGE_ROOT]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
source_summary = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
triage_df = pd.read_csv(EXACT_TRIAGE_PATH)
triage_summary = json.loads(TRIAGE_SUMMARY_PATH.read_text(encoding='utf-8'))

required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_synthstrip_mask_path',
    'trial8_synthstrip_mask_sha256',
    'cow_prior_valid',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        f'Frozen manifest is missing columns: {missing_manifest_columns}'
    )
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError(f'Frozen manifest has {len(manifest_df)} subjects.')
if manifest_df['subject_id'].duplicated().any():
    raise RuntimeError('Frozen manifest contains duplicate subjects.')
if manifest_df['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
    raise RuntimeError('Frozen manifest split counts changed.')
if file_sha256(FROZEN_MANIFEST_PATH) != source_summary.get(
    'frozen_manifest_sha256'
):
    raise RuntimeError('Frozen manifest hash changed after bulk registration.')
if not as_bool(source_summary.get('all_subjects_completed', False)):
    raise RuntimeError('The source bulk registration did not complete all subjects.')
if as_bool(source_summary.get('target_arrays_read', True)):
    raise RuntimeError('The source registration reports target access.')
if int(triage_summary.get('subjects', -1)) != EXPECTED_SUBJECTS:
    raise RuntimeError('The safety triage does not contain 148 subjects.')
if int(triage_summary.get('automated_safety_failed', -1)) != 34:
    raise RuntimeError('The expected 34-case safety triage changed.')

manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
manifest_df = manifest_df.set_index('subject_id', drop=False)
triage_df['subject_id'] = triage_df['subject_id'].astype(str)
triage_df = triage_df.set_index('subject_id', drop=False)

missing_fallback_subjects = sorted(set(FALLBACK_SUBJECTS) - set(manifest_df.index))
if missing_fallback_subjects:
    raise RuntimeError(
        'Fallback subjects missing from manifest: '
        + ', '.join(missing_fallback_subjects)
    )

noncandidate_failed = triage_df.loc[
    (~triage_df['automated_safety_passed'].map(as_bool))
    & (~triage_df.index.isin(FALLBACK_SUBJECTS))
].copy()
expected_advisory_failure = 'raw_atlas_at_least_0p80_inside_brain'
noncandidate_failure_sets = {
    str(value).strip() for value in noncandidate_failed['failed_checks']
}
if noncandidate_failure_sets != {expected_advisory_failure}:
    raise RuntimeError(
        'A noncandidate subject has more than the approved advisory-only '
        f'failure pattern: {sorted(noncandidate_failure_sets)}'
    )
if len(noncandidate_failed) != 25:
    raise RuntimeError(
        f'Expected 25 advisory-only SyN cases, found {len(noncandidate_failed)}.'
    )

jhu_image, jhu_array = load_effective_3d(JHU_REFERENCE_PATH, np.float32)
fine_image, fine_source_labels = load_effective_3d(
    FULL_FINE_ATLAS_PATH,
    np.int16,
)
fine_source_labels = np.rint(fine_source_labels).astype(np.int16)
if jhu_array.shape != fine_source_labels.shape:
    raise RuntimeError('JHU reference and fine atlas shapes differ.')
if not np.allclose(jhu_image.affine, fine_image.affine, atol=1e-5, rtol=1e-5):
    raise RuntimeError('JHU reference and fine atlas affines differ.')
moving_mask_np = jhu_array > 0

schema_df = pd.read_csv(SCHEMA_CSV_PATH)
fine_to_level2 = dict(
    zip(
        schema_df['fine_label'].astype(int),
        schema_df['level2_label'].astype(int),
    )
)

fallback_signature_payload = {
    'code_revision': CODE_REVISION,
    'fallback_version': FALLBACK_VERSION,
    'source_registration_version': SOURCE_REGISTRATION_VERSION,
    'frozen_manifest_sha256': file_sha256(FROZEN_MANIFEST_PATH),
    'source_summary_sha256': file_sha256(SOURCE_SUMMARY_PATH),
    'triage_summary_sha256': file_sha256(TRIAGE_SUMMARY_PATH),
    'jhu_reference_sha256': file_sha256(JHU_REFERENCE_PATH),
    'fine_atlas_sha256': file_sha256(FULL_FINE_ATLAS_PATH),
    'fallback_subjects': FALLBACK_SUBJECTS,
    'advisory_checks': sorted(ADVISORY_CHECKS),
    'selection_policy': (
        'affine only when all final-use safety checks pass; otherwise disabled'
    ),
}
fallback_signature = object_sha256(fallback_signature_payload)

config_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'code_revision': CODE_REVISION,
    'fallback_version': FALLBACK_VERSION,
    'fallback_signature': fallback_signature,
    'fallback_signature_payload': fallback_signature_payload,
    'definite_fallback_subjects': DEFINITE_FALLBACK_SUBJECTS,
    'borderline_fallback_subjects': BORDERLINE_FALLBACK_SUBJECTS,
    'fallback_subjects': FALLBACK_SUBJECTS,
    'selection_policy': {
        'existing_safe_or_advisory_only': 'syn',
        'fallback_affine_all_final_checks_pass': 'affine',
        'fallback_affine_any_final_check_fails': 'disabled',
    },
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'source_registration_outputs_modified': False,
}
atomic_write_json(config_record, CONFIG_PATH)


def validate_source_case(subject_id, verify_all_hashes=False):
    case_dir = SOURCE_CASE_ROOT / subject_id
    complete_path = case_dir / 'complete.json'
    if not complete_path.exists():
        raise FileNotFoundError(complete_path)
    record = json.loads(complete_path.read_text(encoding='utf-8'))
    if str(record.get('subject_id')) != subject_id:
        raise RuntimeError(f'{subject_id} source record mismatch.')
    if record.get('registration_version') != SOURCE_REGISTRATION_VERSION:
        raise RuntimeError(f'{subject_id} source registration version changed.')
    if record.get('completed') is not True:
        raise RuntimeError(f'{subject_id} source registration is incomplete.')
    if record.get('target_arrays_read') is not False:
        raise RuntimeError(f'{subject_id} source registration read targets.')
    output_hashes = record.get('output_sha256', {})
    if not output_hashes:
        raise RuntimeError(f'{subject_id} source hashes are missing.')
    for file_name, expected_hash in output_hashes.items():
        output_path = case_dir / file_name
        if not output_path.exists():
            raise FileNotFoundError(output_path)
        if verify_all_hashes and file_sha256(output_path) != expected_hash:
            raise RuntimeError(f'{subject_id} source output hash changed: {file_name}')
    return case_dir, record


def validate_candidate_case(subject_id, subject_dir):
    complete_path = subject_dir / 'candidate_complete.json'
    if not complete_path.exists():
        return None
    record = json.loads(complete_path.read_text(encoding='utf-8'))
    expected_values = {
        'subject_id': subject_id,
        'fallback_version': FALLBACK_VERSION,
        'fallback_signature': fallback_signature,
        'completed': True,
        'target_arrays_read': False,
    }
    if any(record.get(key) != value for key, value in expected_values.items()):
        return None
    output_hashes = record.get('output_sha256', {})
    if not output_hashes:
        return None
    for file_name, expected_hash in output_hashes.items():
        output_path = subject_dir / file_name
        if not output_path.exists() or file_sha256(output_path) != expected_hash:
            return None
    return record


def quarantine_incomplete(subject_dir):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    target = QUARANTINE_ROOT / f'{subject_dir.name}_{timestamp}'
    counter = 1
    while target.exists():
        target = QUARANTINE_ROOT / f'{subject_dir.name}_{timestamp}_{counter:02d}'
        counter += 1
    subject_dir.replace(target)
    return target


def process_fallback_subject(subject_id):
    manifest_row = manifest_df.loc[subject_id]
    source_case_dir, source_record = validate_source_case(
        subject_id,
        verify_all_hashes=True,
    )
    destination = CASE_ROOT / subject_id

    if destination.exists():
        completed = validate_candidate_case(subject_id, destination)
        if completed is not None:
            print(f'Reused affine candidate: {subject_id}')
            return completed, True
        quarantined = quarantine_incomplete(destination)
        print(f'Quarantined incomplete affine candidate: {quarantined}')

    npz_path = Path(str(manifest_row['vascufate_v2_npz_path']))
    mask_path = Path(str(manifest_row['trial8_synthstrip_mask_path']))
    if not npz_path.exists() or not mask_path.exists():
        raise FileNotFoundError(f'{subject_id} frozen input is missing.')
    if file_sha256(npz_path) != str(manifest_row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id} NPZ hash changed after freeze.')
    if file_sha256(mask_path) != str(
        manifest_row['trial8_synthstrip_mask_sha256']
    ):
        raise RuntimeError(f'{subject_id} SynthStrip hash changed after freeze.')

    forward_files = [
        source_case_dir / str(file_name)
        for file_name in source_record['forward_transform_files']
    ]
    affine_paths = [path for path in forward_files if path.suffix.lower() == '.mat']
    warp_paths = [
        path
        for path in forward_files
        if str(path).lower().endswith(('.nii', '.nii.gz'))
    ]
    if len(affine_paths) != 1 or not warp_paths:
        raise RuntimeError(
            f'{subject_id} expected one affine and at least one warp; found '
            f'{len(affine_paths)} affine and {len(warp_paths)} warp.'
        )
    if any(not path.exists() for path in forward_files):
        raise FileNotFoundError(f'{subject_id} saved transform is missing.')

    local_parent = Path(
        tempfile.mkdtemp(prefix=f'trial8_affine_{subject_id}_', dir='/tmp')
    )
    local_case_dir = local_parent / subject_id
    local_case_dir.mkdir(parents=True, exist_ok=False)
    start_time = time.time()

    try:
        with np.load(npz_path, allow_pickle=False) as data:
            required_keys = {
                'image',
                'crop_mask',
                'cow_label',
                'lvo_proximity',
                'input_channels',
                'affine',
                'subject_id',
            }
            missing_keys = sorted(required_keys - set(data.files))
            if missing_keys:
                raise RuntimeError(f'{subject_id} NPZ missing keys: {missing_keys}')
            if str(data['subject_id'].item()) != subject_id:
                raise RuntimeError(f'{subject_id} NPZ subject mismatch.')
            channel_names = [str(value) for value in data['input_channels']]
            brain_index = channel_names.index('ncct_brain_window')
            broad_index = channel_names.index('ncct_broad_window')
            brain_window = data['image'][brain_index].astype(np.float32)
            broad_window = data['image'][broad_index].astype(np.float32)
            crop_mask_np = data['crop_mask'].astype(bool)
            cow_label = data['cow_label'].astype(np.uint8)
            lvo_proximity = data['lvo_proximity'].astype(np.float32)
            patient_affine = data['affine'].astype(np.float64)

        mask_image, mask_array = load_effective_3d(mask_path, np.float32)
        fixed_mask_np = mask_array > 0.5
        if not (
            brain_window.shape
            == broad_window.shape
            == crop_mask_np.shape
            == cow_label.shape
            == lvo_proximity.shape
            == fixed_mask_np.shape
        ):
            raise RuntimeError(f'{subject_id} spatial shapes differ.')
        if not np.allclose(mask_image.affine, patient_affine, atol=1e-5, rtol=1e-5):
            raise RuntimeError(f'{subject_id} mask affine differs from NPZ.')
        if ''.join(nib.aff2axcodes(patient_affine)) != 'RAS':
            raise RuntimeError(f'{subject_id} is not RAS-oriented.')
        spacing = np.sqrt(np.sum(patient_affine[:3, :3] ** 2, axis=0))
        if not np.allclose(spacing, EXPECTED_SPACING, atol=1e-4, rtol=0):
            raise RuntimeError(f'{subject_id} spacing changed.')

        brain_inside_crop_fraction = float(
            np.logical_and(fixed_mask_np, crop_mask_np).sum()
            / max(fixed_mask_np.sum(), 1)
        )
        fixed_array = np.clip(
            0.75 * brain_window + 0.25 * broad_window,
            0.0,
            1.0,
        ).astype(np.float32)
        fixed_array[~fixed_mask_np] = 0.0

        fixed_path = local_case_dir / 'patient_ncct_fixed.nii.gz'
        moving_path = local_case_dir / 'jhu_mni_ss_moving.nii.gz'
        moving_mask_path = local_case_dir / 'jhu_mni_ss_mask.nii.gz'
        fine_path = local_case_dir / 'arterial_atlas_full_fine_3d.nii.gz'
        save_nifti(fixed_array, patient_affine, fixed_path, np.float32)
        save_nifti(jhu_array, jhu_image.affine, moving_path, np.float32)
        save_nifti(
            moving_mask_np.astype(np.uint8),
            jhu_image.affine,
            moving_mask_path,
            np.uint8,
        )
        save_nifti(fine_source_labels, fine_image.affine, fine_path, np.int16)

        fixed = ants.image_read(str(fixed_path), pixeltype='float')
        moving = ants.image_read(str(moving_path), pixeltype='float')
        moving_mask = ants.image_read(str(moving_mask_path), pixeltype='unsigned char')
        moving_labels = ants.image_read(str(fine_path), pixeltype='float')

        affine_transform_list = [str(affine_paths[0])]
        full_transform_list = [str(path) for path in forward_files]

        affine_warped_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving,
            transformlist=affine_transform_list,
            interpolator='linear',
        )
        affine_mask_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving_mask,
            transformlist=affine_transform_list,
            interpolator='nearestNeighbor',
        )
        affine_labels_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving_labels,
            transformlist=affine_transform_list,
            interpolator='genericLabel',
        )
        syn_warped_image = ants.apply_transforms(
            fixed=fixed,
            moving=moving,
            transformlist=full_transform_list,
            interpolator='linear',
        )

        affine_warped_np = affine_warped_image.numpy().astype(np.float32)
        affine_mask_np = affine_mask_image.numpy() > 0.5
        affine_full_fine_raw = np.rint(
            affine_labels_image.numpy()
        ).astype(np.int16)
        syn_warped_np = syn_warped_image.numpy().astype(np.float32)

        unexpected_labels = sorted(
            set(int(value) for value in np.unique(affine_full_fine_raw))
            - set(range(33))
        )
        affine_full_fine_clipped = affine_full_fine_raw.copy()
        affine_full_fine_clipped[~fixed_mask_np] = 0
        affine_vascular = affine_full_fine_clipped.copy()
        affine_vascular[np.isin(affine_vascular, [31, 32])] = 0
        affine_level2 = np.zeros_like(affine_full_fine_clipped, dtype=np.int16)
        for fine_label, level2_label in fine_to_level2.items():
            affine_level2[affine_full_fine_clipped == fine_label] = level2_label
        affine_valid = affine_full_fine_clipped > 0

        fixed_center = center_of_mass_world(fixed_mask_np, patient_affine)
        affine_center = center_of_mass_world(affine_mask_np, patient_affine)
        affine_center_distance_mm = float(
            np.linalg.norm(fixed_center - affine_center)
        )
        affine_overlap = fixed_mask_np & affine_mask_np
        affine_nmi = normalized_mutual_information(
            fixed_array,
            affine_warped_np,
            affine_overlap,
        )
        affine_mask_dice = dice_binary(fixed_mask_np, affine_mask_np)

        raw_atlas_support = affine_full_fine_raw > 0
        raw_support_count = int(raw_atlas_support.sum())
        raw_inside_count = int(
            np.logical_and(raw_atlas_support, fixed_mask_np).sum()
        )
        raw_inside_fraction = (
            float(raw_inside_count / raw_support_count)
            if raw_support_count
            else 0.0
        )
        full_brain_coverage = float(
            np.logical_and(raw_atlas_support, fixed_mask_np).sum()
            / max(fixed_mask_np.sum(), 1)
        )
        distance_inside_brain = ndimage.distance_transform_edt(
            fixed_mask_np,
            sampling=spacing,
        )
        brain_core = distance_inside_brain >= 5.0
        core_coverage = float(
            np.logical_and(raw_atlas_support, brain_core).sum()
            / max(brain_core.sum(), 1)
        )

        present_vascular_labels = sorted(
            int(value)
            for value in np.unique(affine_vascular)
            if 1 <= int(value) <= 30
        )
        (
            present_homolog_pairs,
            correctly_ordered_pairs,
            homolog_x_differences,
        ) = homolog_pair_metrics(affine_full_fine_clipped, patient_affine)

        affine_transform = ants.read_transform(str(affine_paths[0]))
        affine_parameters = np.asarray(
            affine_transform.parameters,
            dtype=np.float64,
        )
        if affine_parameters.size < 9:
            raise RuntimeError(f'{subject_id} affine parameters are incomplete.')
        affine_matrix = affine_parameters[:9].reshape(3, 3)
        affine_determinant = float(np.linalg.det(affine_matrix))

        affine_checks = {
            'fixed_orientation_is_ras': (
                ''.join(nib.aff2axcodes(patient_affine)) == 'RAS'
            ),
            'brain_mask_at_least_0p99_inside_crop': (
                brain_inside_crop_fraction >= 0.99
            ),
            'no_unexpected_labels': len(unexpected_labels) == 0,
            'at_least_24_of_30_vascular_labels_present': (
                len(present_vascular_labels) >= 24
            ),
            'at_least_12_homolog_pairs_present': (
                present_homolog_pairs >= 12
            ),
            'all_present_pairs_have_correct_left_right_order': (
                correctly_ordered_pairs == present_homolog_pairs
            ),
            'affine_determinant_is_finite_and_positive': (
                np.isfinite(affine_determinant) and affine_determinant > 0
            ),
            'full_brain_coverage_at_least_0p80': (
                full_brain_coverage >= 0.80
            ),
            'brain_core_coverage_at_least_0p85': core_coverage >= 0.85,
            'nonlinear_mask_dice_at_least_0p70': affine_mask_dice >= 0.70,
            'nonlinear_center_distance_at_most_25mm': (
                affine_center_distance_mm <= 25.0
            ),
            'nonlinear_nmi_is_finite': bool(np.isfinite(affine_nmi)),
            'clipped_labels_remain_inside_brain': bool(
                not np.any((affine_full_fine_clipped > 0) & ~fixed_mask_np)
            ),
        }
        affine_advisory_checks = {
            'raw_atlas_at_least_0p80_inside_brain': raw_inside_fraction >= 0.80,
        }
        affine_final_use_safety_passed = bool(all(affine_checks.values()))
        proposed_mode = 'affine' if affine_final_use_safety_passed else 'disabled'

        affine_metrics = {
            'nonlinear_mask_dice': affine_mask_dice,
            'nonlinear_nmi': affine_nmi,
            'nonlinear_center_distance_mm': affine_center_distance_mm,
            'raw_atlas_inside_brain_fraction': raw_inside_fraction,
            'full_brain_coverage_fraction': full_brain_coverage,
            'brain_core_coverage_fraction': core_coverage,
            'present_vascular_label_count': len(present_vascular_labels),
            'present_homolog_pairs': present_homolog_pairs,
            'correctly_ordered_homolog_pairs': correctly_ordered_pairs,
            'minimum_right_minus_left_centroid_mm': (
                float(min(homolog_x_differences))
                if homolog_x_differences
                else None
            ),
            'affine_determinant': affine_determinant,
        }
        syn_metrics = {
            'nonlinear_mask_dice': float(source_record['nonlinear_mask_dice']),
            'nonlinear_nmi': float(source_record['nonlinear_nmi']),
            'nonlinear_center_distance_mm': float(
                source_record['nonlinear_center_distance_mm']
            ),
            'raw_atlas_inside_brain_fraction': float(
                source_record['raw_atlas_inside_brain_fraction']
            ),
            'full_brain_coverage_fraction': float(
                source_record['full_brain_coverage_fraction']
            ),
            'brain_core_coverage_fraction': float(
                source_record['brain_core_coverage_fraction']
            ),
            'present_vascular_label_count': int(
                source_record['present_vascular_label_count']
            ),
            'present_homolog_pairs': int(source_record['present_homolog_pairs']),
            'correctly_ordered_homolog_pairs': int(
                source_record['correctly_ordered_homolog_pairs']
            ),
        }

        affine_fine_path = (
            local_case_dir / 'arterial_atlas_full_fine_affine_clipped.nii.gz'
        )
        affine_vascular_path = (
            local_case_dir / 'arterial_atlas_vascular30_affine_clipped.nii.gz'
        )
        affine_level2_path = (
            local_case_dir / 'arterial_atlas_level2_affine_clipped.nii.gz'
        )
        affine_valid_path = local_case_dir / 'atlas_valid_mask_affine.nii.gz'
        affine_warped_path = local_case_dir / 'affine_warped_jhu.nii.gz'
        affine_copy_path = local_case_dir / 'source_affine_transform.mat'
        comparison_qa_path = local_case_dir / 'syn_vs_affine_comparison_qa.png'

        save_nifti(
            affine_full_fine_clipped,
            patient_affine,
            affine_fine_path,
            np.int16,
        )
        save_nifti(
            affine_vascular,
            patient_affine,
            affine_vascular_path,
            np.int16,
        )
        save_nifti(
            affine_level2,
            patient_affine,
            affine_level2_path,
            np.int16,
        )
        save_nifti(
            affine_valid.astype(np.uint8),
            patient_affine,
            affine_valid_path,
            np.uint8,
        )
        save_nifti(
            affine_warped_np,
            patient_affine,
            affine_warped_path,
            np.float32,
        )
        shutil.copy2(affine_paths[0], affine_copy_path)

        syn_labels_path = source_case_dir / source_record['fine_labels_file']
        _, syn_labels = load_effective_3d(syn_labels_path, np.int16)
        save_comparison_qa(
            fixed_array=fixed_array,
            fixed_mask=fixed_mask_np,
            syn_warped=syn_warped_np,
            affine_warped=affine_warped_np,
            syn_labels=syn_labels,
            affine_labels=affine_full_fine_clipped,
            cow_label=cow_label,
            lvo_proximity=lvo_proximity,
            output_path=comparison_qa_path,
            subject_id=subject_id,
            syn_metrics=syn_metrics,
            affine_metrics=affine_metrics,
            recommendation=proposed_mode,
        )

        for temporary_path in [
            fixed_path,
            moving_path,
            moving_mask_path,
            fine_path,
        ]:
            if temporary_path.exists():
                temporary_path.unlink()

        output_sha256 = {
            path.name: file_sha256(path)
            for path in sorted(local_case_dir.iterdir())
            if path.is_file()
        }
        elapsed_seconds = float(time.time() - start_time)
        record = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'completed': True,
            'code_revision': CODE_REVISION,
            'fallback_version': FALLBACK_VERSION,
            'fallback_signature': fallback_signature,
            'subject_id': subject_id,
            'dev_split': str(manifest_row['dev_split']),
            'fallback_scope': (
                'borderline'
                if subject_id in BORDERLINE_FALLBACK_SUBJECTS
                else 'definite'
            ),
            'source_registration_version': SOURCE_REGISTRATION_VERSION,
            'source_complete_path': str(source_case_dir / 'complete.json'),
            'source_affine_transform_path': str(affine_paths[0]),
            'source_affine_transform_sha256': file_sha256(affine_paths[0]),
            'npz_path': str(npz_path),
            'npz_sha256': str(manifest_row['trial8_npz_sha256']),
            'synthstrip_mask_path': str(mask_path),
            'synthstrip_mask_sha256': str(
                manifest_row['trial8_synthstrip_mask_sha256']
            ),
            'cow_prior_valid': as_bool(manifest_row['cow_prior_valid']),
            'syn_metrics': syn_metrics,
            'syn_final_use_checks': final_use_checks_from_syn(source_record),
            'syn_final_use_safety_passed': bool(
                all(final_use_checks_from_syn(source_record).values())
            ),
            'affine_metrics': affine_metrics,
            'affine_final_use_checks': affine_checks,
            'affine_advisory_checks': affine_advisory_checks,
            'affine_final_use_safety_passed': affine_final_use_safety_passed,
            'proposed_mode': proposed_mode,
            'proposed_atlas_enabled': proposed_mode == 'affine',
            'proposed_registration_confidence': (
                quality_confidence(affine_metrics)
                if proposed_mode == 'affine'
                else 0.0
            ),
            'fine_labels_file': affine_fine_path.name,
            'vascular30_file': affine_vascular_path.name,
            'level2_file': affine_level2_path.name,
            'atlas_valid_mask_file': affine_valid_path.name,
            'affine_warped_jhu_file': affine_warped_path.name,
            'affine_transform_file': affine_copy_path.name,
            'comparison_qa_file': comparison_qa_path.name,
            'elapsed_seconds': elapsed_seconds,
            'target_arrays_read': False,
            'study_trainable_parameters': 0,
            'source_registration_outputs_modified': False,
            'candidate_outputs_frozen': False,
            'model_training_authorized': False,
            'output_sha256': output_sha256,
        }
        atomic_write_json(record, local_case_dir / 'candidate_complete.json')
        shutil.copytree(local_case_dir, destination)

        committed = validate_candidate_case(subject_id, destination)
        if committed is None:
            quarantine_incomplete(destination)
            raise RuntimeError(f'{subject_id} candidate failed commit validation.')

        print(
            f'Completed {subject_id}: proposal={proposed_mode}, '
            f'affineSafety={affine_final_use_safety_passed}, '
            f'Dice={affine_mask_dice:.4f}, '
            f'full={full_brain_coverage:.4f}, '
            f'core={core_coverage:.4f}, '
            f'center={affine_center_distance_mm:.1f}mm, '
            f'{elapsed_seconds / 60:.1f} min'
        )
        return committed, False
    finally:
        if local_parent.exists():
            shutil.rmtree(local_parent)


print('Trial8 affine fallback candidate evaluation')
print(f'Fallback version: {FALLBACK_VERSION}')
print(f'Candidates: {FALLBACK_SUBJECTS}')
print(f'ANTsPy version: {getattr(ants, "__version__", "unknown")}')
print(f'SciPy version: {scipy.__version__}')
print(f'ITK threads: {ITK_THREADS}')
print('Registration compute: CPU transform application only')
print('No registration optimization will be repeated.')
print('Target arrays read: False')
print('Study trainable parameters: 0 (fallback preprocessing only)')

candidate_records = []
computed_count = 0
reused_count = 0
for index, subject_id in enumerate(FALLBACK_SUBJECTS, 1):
    print('\n' + '=' * 80)
    print(f'Affine fallback {index}/{len(FALLBACK_SUBJECTS)}: {subject_id}')
    print('=' * 80)
    candidate_record, reused = process_fallback_subject(subject_id)
    candidate_records.append(candidate_record)
    if reused:
        reused_count += 1
    else:
        computed_count += 1

comparison_rows = []
for record in candidate_records:
    syn = record['syn_metrics']
    affine = record['affine_metrics']
    failed_affine_checks = sorted(
        name
        for name, passed in record['affine_final_use_checks'].items()
        if not as_bool(passed)
    )
    comparison_rows.append({
        'subject_id': record['subject_id'],
        'dev_split': record['dev_split'],
        'fallback_scope': record['fallback_scope'],
        'syn_mask_dice': syn['nonlinear_mask_dice'],
        'affine_mask_dice': affine['nonlinear_mask_dice'],
        'dice_change_affine_minus_syn': (
            affine['nonlinear_mask_dice'] - syn['nonlinear_mask_dice']
        ),
        'syn_full_brain_coverage': syn['full_brain_coverage_fraction'],
        'affine_full_brain_coverage': affine['full_brain_coverage_fraction'],
        'full_coverage_change_affine_minus_syn': (
            affine['full_brain_coverage_fraction']
            - syn['full_brain_coverage_fraction']
        ),
        'syn_core_coverage': syn['brain_core_coverage_fraction'],
        'affine_core_coverage': affine['brain_core_coverage_fraction'],
        'core_coverage_change_affine_minus_syn': (
            affine['brain_core_coverage_fraction']
            - syn['brain_core_coverage_fraction']
        ),
        'syn_center_distance_mm': syn['nonlinear_center_distance_mm'],
        'affine_center_distance_mm': affine['nonlinear_center_distance_mm'],
        'center_change_affine_minus_syn_mm': (
            affine['nonlinear_center_distance_mm']
            - syn['nonlinear_center_distance_mm']
        ),
        'affine_raw_atlas_inside_brain_fraction': (
            affine['raw_atlas_inside_brain_fraction']
        ),
        'affine_vascular_labels': affine['present_vascular_label_count'],
        'affine_homolog_pairs': affine['present_homolog_pairs'],
        'affine_correctly_ordered_pairs': (
            affine['correctly_ordered_homolog_pairs']
        ),
        'affine_determinant': affine['affine_determinant'],
        'affine_final_use_safety_passed': (
            record['affine_final_use_safety_passed']
        ),
        'affine_failed_checks': '; '.join(failed_affine_checks),
        'proposed_mode': record['proposed_mode'],
        'proposed_registration_confidence': (
            record['proposed_registration_confidence']
        ),
        'comparison_qa_path': str(
            CASE_ROOT
            / record['subject_id']
            / record['comparison_qa_file']
        ),
        'target_arrays_read': False,
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('subject_id')
atomic_write_csv(comparison_df, COMPARISON_AUDIT_PATH)

candidate_by_subject = {
    record['subject_id']: record for record in candidate_records
}
selection_rows = []
for subject_id, manifest_row in manifest_df.sort_index().iterrows():
    source_case_dir, source_record = validate_source_case(
        subject_id,
        verify_all_hashes=False,
    )
    syn_final_checks = final_use_checks_from_syn(source_record)
    syn_final_safe = bool(all(syn_final_checks.values()))

    if subject_id in candidate_by_subject:
        candidate = candidate_by_subject[subject_id]
        selected_mode = candidate['proposed_mode']
        atlas_enabled = selected_mode == 'affine'
        confidence = float(candidate['proposed_registration_confidence'])
        if atlas_enabled:
            selected_case_dir = CASE_ROOT / subject_id
            fine_path = selected_case_dir / candidate['fine_labels_file']
            vascular_path = selected_case_dir / candidate['vascular30_file']
            level2_path = selected_case_dir / candidate['level2_file']
            valid_path = selected_case_dir / candidate['atlas_valid_mask_file']
            metrics_path = selected_case_dir / 'candidate_complete.json'
            selection_reason = 'affine_passed_all_final_use_checks'
        else:
            fine_path = ''
            vascular_path = ''
            level2_path = ''
            valid_path = ''
            metrics_path = CASE_ROOT / subject_id / 'candidate_complete.json'
            selection_reason = 'syn_and_affine_failed_final_use_checks'
    else:
        if not syn_final_safe:
            raise RuntimeError(
                f'{subject_id} is outside fallback scope but fails final-use safety.'
            )
        selected_mode = 'syn'
        atlas_enabled = True
        confidence = quality_confidence(source_record)
        fine_path = source_case_dir / source_record['fine_labels_file']
        vascular_path = source_case_dir / source_record['vascular30_file']
        level2_path = source_case_dir / source_record['level2_file']
        valid_path = source_case_dir / source_record['atlas_valid_mask_file']
        metrics_path = source_case_dir / 'complete.json'
        source_checks = {
            str(name): as_bool(value)
            for name, value in source_record['automated_safety_checks'].items()
        }
        selection_reason = (
            'syn_passed_all_original_checks'
            if all(source_checks.values())
            else 'syn_failed_only_advisory_raw_support_check'
        )

    selection_rows.append({
        'subject_id': subject_id,
        'dev_split': str(manifest_row['dev_split']),
        'cow_prior_valid': as_bool(manifest_row['cow_prior_valid']),
        'selected_mode': selected_mode,
        'atlas_enabled': atlas_enabled,
        'registration_confidence': confidence,
        'selection_reason': selection_reason,
        'selected_fine_labels_path': str(fine_path),
        'selected_vascular30_path': str(vascular_path),
        'selected_level2_path': str(level2_path),
        'selected_atlas_valid_mask_path': str(valid_path),
        'selected_metrics_path': str(metrics_path),
        'target_arrays_read': False,
        'candidate_policy_frozen': False,
        'model_training_authorized': False,
    })

selection_df = pd.DataFrame(selection_rows).sort_values('subject_id')
if len(selection_df) != EXPECTED_SUBJECTS:
    raise RuntimeError('Candidate selection manifest does not contain 148 rows.')
if selection_df['subject_id'].nunique() != EXPECTED_SUBJECTS:
    raise RuntimeError('Candidate selection manifest contains duplicate subjects.')
if selection_df['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
    raise RuntimeError('Candidate selection split counts changed.')
if selection_df['target_arrays_read'].any():
    raise RuntimeError('Candidate selection unexpectedly reports target access.')

atomic_write_csv(selection_df, SELECTION_MANIFEST_PATH)
montage_paths = make_montages(
    comparison_df.to_dict('records'),
    MONTAGE_ROOT,
)

mode_counts = selection_df['selected_mode'].value_counts().to_dict()
summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'code_revision': CODE_REVISION,
    'fallback_version': FALLBACK_VERSION,
    'fallback_signature': fallback_signature,
    'subjects': EXPECTED_SUBJECTS,
    'fallback_candidates': len(FALLBACK_SUBJECTS),
    'computed_this_run': computed_count,
    'reused_this_run': reused_count,
    'proposed_mode_counts': {
        'syn': int(mode_counts.get('syn', 0)),
        'affine': int(mode_counts.get('affine', 0)),
        'disabled': int(mode_counts.get('disabled', 0)),
    },
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'source_registration_outputs_modified': False,
    'fallback_candidates_frozen': False,
    'manual_visual_qa_approved': False,
    'model_training_authorized': False,
    'comparison_audit_path': str(COMPARISON_AUDIT_PATH),
    'candidate_selection_manifest_path': str(SELECTION_MANIFEST_PATH),
    'comparison_montage_paths': [str(path) for path in montage_paths],
}
atomic_write_json(summary, SUMMARY_PATH)

print('\nTrial8 SyN versus affine fallback comparison:')
display(comparison_df)

print('\nProposed atlas mode counts by split:')
mode_split_df = selection_df.groupby(
    ['dev_split', 'selected_mode'],
    as_index=False,
).agg(subjects=('subject_id', 'count'))
display(mode_split_df)

print('\nAffine fallback comparison QA:')
for montage_path in montage_paths:
    print(montage_path)
    display(IPythonImage(filename=str(montage_path)))

print('\n' + '=' * 80)
print('TRIAL8 AFFINE FALLBACK CANDIDATES COMPLETE')
print('=' * 80)
print(f'Fallback candidates:                 {len(FALLBACK_SUBJECTS)}')
print(f'Computed this run:                   {computed_count}')
print(f'Reused this run:                     {reused_count}')
print(f'Proposed SyN subjects:               {mode_counts.get("syn", 0)}')
print(f'Proposed affine subjects:            {mode_counts.get("affine", 0)}')
print(f'Proposed atlas-disabled subjects:    {mode_counts.get("disabled", 0)}')
print('Target arrays read:                  False')
print('Study trainable parameters:          0 (fallback preprocessing only)')
print('Source SyN outputs modified:         False')
print('Fallback candidates frozen:          False')
print('Model training authorized:           False')
print(f'Comparison audit:                    {COMPARISON_AUDIT_PATH}')
print(f'Candidate selection manifest:        {SELECTION_MANIFEST_PATH}')
print(f'QA directory:                        {MONTAGE_ROOT}')
print(f'Summary:                             {SUMMARY_PATH}')
print('\nNext: check the nine comparison panels before freezing the policy.')


In [ ]:
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from google.colab import drive
from IPython.display import display


drive.mount('/mnt/storage')

PROJECT_ROOT = Path('/path/to/project')
FROZEN_PREPROCESSING_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
SOURCE_REGISTRATION_VERSION = 'v1_jhu_to_ncct_synra_synthstrip_r2_all148'
TRIAGE_VERSION = 'v1_exact_checks_and_visual_qa'
FALLBACK_VERSION = 'v1_syn_affine_or_disabled_candidate'
POLICY_VERSION = 'v1_syn139_affine2_disabled7_frozen'
CODE_REVISION = 'r1_final_atlas_policy_freeze'

FROZEN_DATASET_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v2_vascufate_r2_frozen.csv'
)
SOURCE_REGISTRATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / FROZEN_PREPROCESSING_VERSION
    / 'vascular_atlas'
    / 'bulk_registration'
    / SOURCE_REGISTRATION_VERSION
)
SOURCE_CASE_ROOT = SOURCE_REGISTRATION_ROOT / 'cases'
SOURCE_SUMMARY_PATH = SOURCE_REGISTRATION_ROOT / 'bulk_registration_summary.json'

TRIAGE_ROOT = SOURCE_REGISTRATION_ROOT / 'safety_triage' / TRIAGE_VERSION
TRIAGE_SUMMARY_PATH = TRIAGE_ROOT / 'safety_triage_summary.json'

FALLBACK_ROOT = (
    SOURCE_REGISTRATION_ROOT
    / 'fallback_candidates'
    / FALLBACK_VERSION
)
FALLBACK_CASE_ROOT = FALLBACK_ROOT / 'cases'
FALLBACK_SUMMARY_PATH = FALLBACK_ROOT / 'fallback_candidate_summary.json'
CANDIDATE_SELECTION_PATH = FALLBACK_ROOT / 'candidate_selection_manifest.csv'
COMPARISON_AUDIT_PATH = FALLBACK_ROOT / 'affine_candidate_comparison.csv'

OUTPUT_ROOT = (
    SOURCE_REGISTRATION_ROOT
    / 'policy_freeze'
    / POLICY_VERSION
)
POLICY_AUDIT_PATH = OUTPUT_ROOT / 'atlas_policy_integrity_audit.csv'
POLICY_CONFIG_PATH = OUTPUT_ROOT / 'atlas_policy_config.json'
POLICY_GATE_PATH = OUTPUT_ROOT / 'atlas_policy_freeze_gate.json'
FROZEN_MODEL_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_atlas_policy_frozen.csv'
)

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_MODE_COUNTS = {'syn': 139, 'affine': 2, 'disabled': 7}

AFFINE_SUBJECTS = ['sub-stroke0082', 'sub-stroke0162']
DISABLED_SUBJECTS = [
    'sub-stroke0015',
    'sub-stroke0079',
    'sub-stroke0089',
    'sub-stroke0094',
    'sub-stroke0141',
    'sub-stroke0149',
    'sub-stroke0157',
]


MANUAL_AFFINE_QA_APPROVAL = {
    'sub-stroke0082': {
        'approved': True,
        'reviewed_at_utc': '2026-09-19',
        'observation': 'Affine comparison looked much cleaner than SyN.',
    },
    'sub-stroke0162': {
        'approved': True,
        'reviewed_at_utc': '2026-09-19',
        'observation': 'Affine comparison looked much cleaner than SyN.',
    },
}

ADVISORY_CHECKS = {'raw_atlas_at_least_0p80_inside_brain'}


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes'}:
        return True
    if normalized in {'false', '0', 'no'}:
        return False
    raise ValueError(f'Cannot interpret as bool: {value!r}')


def clean_optional_path(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    normalized = str(value).strip()
    return Path(normalized) if normalized else None


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value):
    
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    return value


def atomic_write_json(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + '.tmp')
    temporary_path.write_text(
        json.dumps(json_safe(payload), indent=2, allow_nan=False),
        encoding='utf-8',
    )
    temporary_path.replace(path)


def atomic_write_csv(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)


def load_effective_3d(path, dtype=None):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    if array.ndim != 3:
        raise RuntimeError(f'{path} is not effectively 3D: {array.shape}')
    if dtype is not None:
        array = np.asarray(array, dtype=dtype)
    return image, array


def integer_like(array, tolerance=1e-5):
    array = np.asarray(array)
    if not np.isfinite(array).all():
        return False
    return bool(np.max(np.abs(array - np.rint(array))) <= tolerance)


def final_use_checks_from_syn(record):
    checks = {
        str(name): as_bool(value)
        for name, value in record['automated_safety_checks'].items()
    }
    return {
        name: value
        for name, value in checks.items()
        if name not in ADVISORY_CHECKS
    }


def verify_expected_file(path, output_hashes):
    path = Path(path)
    if not path.exists():
        return False, None
    expected_hash = output_hashes.get(path.name)
    if expected_hash is None:
        return False, None
    observed_hash = file_sha256(path)
    return observed_hash == expected_hash, observed_hash


required_paths = [
    FROZEN_DATASET_MANIFEST_PATH,
    SOURCE_REGISTRATION_ROOT,
    SOURCE_CASE_ROOT,
    SOURCE_SUMMARY_PATH,
    TRIAGE_SUMMARY_PATH,
    FALLBACK_ROOT,
    FALLBACK_CASE_ROOT,
    FALLBACK_SUMMARY_PATH,
    CANDIDATE_SELECTION_PATH,
    COMPARISON_AUDIT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 atlas-policy freeze prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

dataset_manifest_df = pd.read_csv(FROZEN_DATASET_MANIFEST_PATH)
candidate_df = pd.read_csv(CANDIDATE_SELECTION_PATH)
comparison_df = pd.read_csv(COMPARISON_AUDIT_PATH)
source_summary = json.loads(SOURCE_SUMMARY_PATH.read_text(encoding='utf-8'))
triage_summary = json.loads(TRIAGE_SUMMARY_PATH.read_text(encoding='utf-8'))
fallback_summary = json.loads(FALLBACK_SUMMARY_PATH.read_text(encoding='utf-8'))

for frame in [dataset_manifest_df, candidate_df, comparison_df]:
    if 'subject_id' not in frame.columns:
        raise RuntimeError('A policy input is missing subject_id.')
    frame['subject_id'] = frame['subject_id'].astype(str)

required_candidate_columns = {
    'subject_id',
    'dev_split',
    'selected_mode',
    'atlas_enabled',
    'registration_confidence',
    'selection_reason',
    'selected_fine_labels_path',
    'selected_vascular30_path',
    'selected_level2_path',
    'selected_atlas_valid_mask_path',
    'selected_metrics_path',
    'target_arrays_read',
    'candidate_policy_frozen',
    'model_training_authorized',
}
missing_candidate_columns = sorted(
    required_candidate_columns - set(candidate_df.columns)
)
if missing_candidate_columns:
    raise RuntimeError(
        f'Candidate selection is missing columns: {missing_candidate_columns}'
    )

input_checks = {
    'dataset_manifest_has_148_subjects': (
        len(dataset_manifest_df) == EXPECTED_SUBJECTS
    ),
    'dataset_manifest_subjects_are_unique': (
        dataset_manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'candidate_manifest_has_148_subjects': len(candidate_df) == EXPECTED_SUBJECTS,
    'candidate_subjects_are_unique': (
        candidate_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'candidate_subject_set_matches_dataset': (
        set(candidate_df['subject_id']) == set(dataset_manifest_df['subject_id'])
    ),
    'candidate_split_counts_match': (
        candidate_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
    'candidate_mode_counts_match': (
        candidate_df['selected_mode'].value_counts().to_dict()
        == EXPECTED_MODE_COUNTS
    ),
    'affine_subjects_match': (
        sorted(
            candidate_df.loc[
                candidate_df['selected_mode'] == 'affine',
                'subject_id',
            ].tolist()
        )
        == AFFINE_SUBJECTS
    ),
    'disabled_subjects_match': (
        sorted(
            candidate_df.loc[
                candidate_df['selected_mode'] == 'disabled',
                'subject_id',
            ].tolist()
        )
        == DISABLED_SUBJECTS
    ),
    'candidate_reports_no_target_access': (
        not candidate_df['target_arrays_read'].map(as_bool).any()
    ),
    'candidate_policy_is_not_already_frozen': (
        not candidate_df['candidate_policy_frozen'].map(as_bool).any()
    ),
    'candidate_does_not_authorize_training': (
        not candidate_df['model_training_authorized'].map(as_bool).any()
    ),
    'source_registration_completed': as_bool(
        source_summary.get('all_subjects_completed', False)
    ),
    'source_registration_target_free': not as_bool(
        source_summary.get('target_arrays_read', True)
    ),
    'triage_has_148_subjects': int(triage_summary.get('subjects', -1)) == 148,
    'fallback_has_nine_candidates': (
        int(fallback_summary.get('fallback_candidates', -1)) == 9
    ),
    'fallback_target_free': not as_bool(
        fallback_summary.get('target_arrays_read', True)
    ),
    'fallback_not_already_frozen': not as_bool(
        fallback_summary.get('fallback_candidates_frozen', True)
    ),
    'manual_affine_qa_subjects_match': (
        sorted(MANUAL_AFFINE_QA_APPROVAL) == AFFINE_SUBJECTS
    ),
    'manual_affine_qa_all_approved': all(
        as_bool(record['approved'])
        for record in MANUAL_AFFINE_QA_APPROVAL.values()
    ),
}

if not all(input_checks.values()):
    failed = [name for name, passed in input_checks.items() if not passed]
    raise RuntimeError(
        'Trial8 atlas-policy freeze input checks failed: ' + ', '.join(failed)
    )

dataset_manifest_df = dataset_manifest_df.set_index('subject_id', drop=False)
candidate_df = candidate_df.set_index('subject_id', drop=False)
comparison_df = comparison_df.set_index('subject_id', drop=False)

audit_rows = []
integrity_errors = []

for case_index, subject_id in enumerate(sorted(candidate_df.index), 1):
    candidate_row = candidate_df.loc[subject_id]
    dataset_row = dataset_manifest_df.loc[subject_id]
    mode = str(candidate_row['selected_mode'])
    atlas_enabled = as_bool(candidate_row['atlas_enabled'])
    confidence = float(candidate_row['registration_confidence'])
    mask_path = Path(str(dataset_row['trial8_synthstrip_mask_path']))

    row_checks = {
        'split_matches_dataset': (
            str(candidate_row['dev_split']) == str(dataset_row['dev_split'])
        ),
        'mask_exists': mask_path.exists(),
        'mode_is_valid': mode in {'syn', 'affine', 'disabled'},
        'enabled_matches_mode': atlas_enabled == (mode != 'disabled'),
        'confidence_is_finite': bool(np.isfinite(confidence)),
        'confidence_in_unit_interval': 0.0 <= confidence <= 1.0,
        'enabled_confidence_positive': (
            confidence > 0.0 if atlas_enabled else confidence == 0.0
        ),
    }

    selected_paths = {
        'fine': clean_optional_path(candidate_row['selected_fine_labels_path']),
        'vascular': clean_optional_path(candidate_row['selected_vascular30_path']),
        'level2': clean_optional_path(candidate_row['selected_level2_path']),
        'valid': clean_optional_path(
            candidate_row['selected_atlas_valid_mask_path']
        ),
        'metrics': clean_optional_path(candidate_row['selected_metrics_path']),
    }

    manual_qa_required = mode == 'affine'
    manual_qa_approved = (
        as_bool(MANUAL_AFFINE_QA_APPROVAL[subject_id]['approved'])
        if manual_qa_required
        else True
    )
    row_checks['manual_visual_qa_approved_when_required'] = manual_qa_approved

    selected_hashes = {}
    geometry_checks = {}
    label_checks = {}
    metrics_sha256 = None

    if mode == 'disabled':
        row_checks['disabled_paths_are_empty'] = all(
            selected_paths[name] is None
            for name in ['fine', 'vascular', 'level2', 'valid']
        )
        row_checks['disabled_candidate_record_exists'] = (
            selected_paths['metrics'] is not None
            and selected_paths['metrics'].exists()
        )
        if row_checks['disabled_candidate_record_exists']:
            disabled_record = json.loads(
                selected_paths['metrics'].read_text(encoding='utf-8')
            )
            row_checks['disabled_record_subject_matches'] = (
                disabled_record.get('subject_id') == subject_id
            )
            row_checks['disabled_record_proposal_matches'] = (
                disabled_record.get('proposed_mode') == 'disabled'
            )
            row_checks['disabled_affine_safety_failed'] = not as_bool(
                disabled_record.get('affine_final_use_safety_passed', True)
            )
            row_checks['disabled_record_target_free'] = not as_bool(
                disabled_record.get('target_arrays_read', True)
            )
            metrics_sha256 = file_sha256(selected_paths['metrics'])
    else:
        row_checks['enabled_paths_are_complete'] = all(
            selected_paths[name] is not None
            for name in ['fine', 'vascular', 'level2', 'valid', 'metrics']
        )
        row_checks['enabled_paths_exist'] = (
            row_checks['enabled_paths_are_complete']
            and all(path.exists() for path in selected_paths.values())
        )

        if row_checks['enabled_paths_exist']:
            metrics_record = json.loads(
                selected_paths['metrics'].read_text(encoding='utf-8')
            )
            metrics_sha256 = file_sha256(selected_paths['metrics'])
            row_checks['metrics_subject_matches'] = (
                metrics_record.get('subject_id') == subject_id
            )
            row_checks['metrics_target_free'] = not as_bool(
                metrics_record.get('target_arrays_read', True)
            )

            if mode == 'syn':
                expected_case_root = SOURCE_CASE_ROOT / subject_id
                expected_paths = {
                    'fine': expected_case_root / metrics_record['fine_labels_file'],
                    'vascular': expected_case_root / metrics_record['vascular30_file'],
                    'level2': expected_case_root / metrics_record['level2_file'],
                    'valid': expected_case_root / metrics_record['atlas_valid_mask_file'],
                }
                output_hashes = metrics_record.get('output_sha256', {})
                syn_final_checks = final_use_checks_from_syn(metrics_record)
                row_checks['syn_final_use_safety_passed'] = bool(
                    all(syn_final_checks.values())
                )
                row_checks['syn_registration_version_matches'] = (
                    metrics_record.get('registration_version')
                    == SOURCE_REGISTRATION_VERSION
                )
            else:
                expected_case_root = FALLBACK_CASE_ROOT / subject_id
                expected_paths = {
                    'fine': expected_case_root / metrics_record['fine_labels_file'],
                    'vascular': expected_case_root / metrics_record['vascular30_file'],
                    'level2': expected_case_root / metrics_record['level2_file'],
                    'valid': expected_case_root / metrics_record['atlas_valid_mask_file'],
                }
                output_hashes = metrics_record.get('output_sha256', {})
                row_checks['affine_final_use_safety_passed'] = as_bool(
                    metrics_record.get('affine_final_use_safety_passed', False)
                )
                row_checks['affine_proposal_matches'] = (
                    metrics_record.get('proposed_mode') == 'affine'
                )
                row_checks['fallback_version_matches'] = (
                    metrics_record.get('fallback_version') == FALLBACK_VERSION
                )

            row_checks['selected_paths_match_committed_record'] = all(
                selected_paths[name] == expected_paths[name]
                for name in expected_paths
            )

            hash_results = {}
            for name in ['fine', 'vascular', 'level2', 'valid']:
                hash_ok, observed_hash = verify_expected_file(
                    selected_paths[name],
                    output_hashes,
                )
                hash_results[name] = hash_ok
                selected_hashes[name] = observed_hash
            row_checks['selected_artifact_hashes_match'] = all(
                hash_results.values()
            )

            mask_image, mask_array = load_effective_3d(mask_path, np.uint8)
            fine_image, fine_array = load_effective_3d(
                selected_paths['fine'],
                np.float32,
            )
            vascular_image, vascular_array = load_effective_3d(
                selected_paths['vascular'],
                np.float32,
            )
            level2_image, level2_array = load_effective_3d(
                selected_paths['level2'],
                np.float32,
            )
            valid_image, valid_array = load_effective_3d(
                selected_paths['valid'],
                np.uint8,
            )

            arrays = [fine_array, vascular_array, level2_array, valid_array]
            images = [fine_image, vascular_image, level2_image, valid_image]
            geometry_checks = {
                'selected_shapes_match_mask': all(
                    array.shape == mask_array.shape for array in arrays
                ),
                'selected_affines_match_mask': all(
                    np.allclose(
                        image.affine,
                        mask_image.affine,
                        atol=1e-5,
                        rtol=1e-5,
                    )
                    for image in images
                ),
                'selected_orientation_is_ras': all(
                    ''.join(nib.aff2axcodes(image.affine)) == 'RAS'
                    for image in images
                ),
            }

            rounded_fine = np.rint(fine_array).astype(np.int16)
            rounded_vascular = np.rint(vascular_array).astype(np.int16)
            rounded_level2 = np.rint(level2_array).astype(np.int16)
            valid_bool = valid_array > 0
            mask_bool = mask_array > 0
            vascular_labels = {
                int(value)
                for value in np.unique(rounded_vascular)
                if int(value) > 0
            }
            label_checks = {
                'fine_labels_are_integer_like': integer_like(fine_array),
                'vascular_labels_are_integer_like': integer_like(
                    vascular_array
                ),
                'level2_labels_are_integer_like': integer_like(level2_array),
                'fine_labels_in_range_0_to_32': (
                    rounded_fine.min() >= 0 and rounded_fine.max() <= 32
                ),
                'vascular_labels_in_range_0_to_30': (
                    rounded_vascular.min() >= 0
                    and rounded_vascular.max() <= 30
                ),
                'level2_labels_in_range_0_to_10': (
                    rounded_level2.min() >= 0 and rounded_level2.max() <= 10
                ),
                'atlas_valid_is_binary': set(np.unique(valid_array)).issubset(
                    {0, 1}
                ),
                'atlas_valid_matches_fine_support': np.array_equal(
                    valid_bool,
                    rounded_fine > 0,
                ),
                'selected_labels_are_inside_brain_mask': not np.any(
                    (rounded_fine > 0) & ~mask_bool
                ),
                'all_30_vascular_labels_present': (
                    vascular_labels == set(range(1, 31))
                ),
            }
            row_checks['geometry_checks_passed'] = all(
                geometry_checks.values()
            )
            row_checks['label_checks_passed'] = all(label_checks.values())

    failed_checks = [name for name, passed in row_checks.items() if not passed]
    case_integrity_passed = not failed_checks
    if not case_integrity_passed:
        integrity_errors.append(
            f'{subject_id}: ' + ', '.join(failed_checks)
        )

    audit_row = {
        'subject_id': subject_id,
        'dev_split': str(candidate_row['dev_split']),
        'selected_mode': mode,
        'atlas_enabled': atlas_enabled,
        'registration_confidence': confidence,
        'selection_reason': str(candidate_row['selection_reason']),
        'manual_visual_qa_required': manual_qa_required,
        'manual_visual_qa_approved': manual_qa_approved,
        'selected_fine_labels_path': (
            str(selected_paths['fine']) if selected_paths['fine'] else ''
        ),
        'selected_vascular30_path': (
            str(selected_paths['vascular']) if selected_paths['vascular'] else ''
        ),
        'selected_level2_path': (
            str(selected_paths['level2']) if selected_paths['level2'] else ''
        ),
        'selected_atlas_valid_mask_path': (
            str(selected_paths['valid']) if selected_paths['valid'] else ''
        ),
        'selected_metrics_path': (
            str(selected_paths['metrics']) if selected_paths['metrics'] else ''
        ),
        'selected_metrics_sha256': metrics_sha256 or '',
        'selected_fine_sha256': selected_hashes.get('fine') or '',
        'selected_vascular30_sha256': selected_hashes.get('vascular') or '',
        'selected_level2_sha256': selected_hashes.get('level2') or '',
        'selected_atlas_valid_sha256': selected_hashes.get('valid') or '',
        'case_integrity_passed': case_integrity_passed,
        'failed_integrity_checks': '; '.join(failed_checks),
        'target_arrays_read': False,
    }
    audit_rows.append(audit_row)

    if case_index == 1 or case_index % 20 == 0 or case_index == EXPECTED_SUBJECTS:
        print(
            f'Checked {case_index}/{EXPECTED_SUBJECTS}: '
            f'{subject_id}, mode={mode}, integrity={case_integrity_passed}'
        )

audit_df = pd.DataFrame(audit_rows).sort_values('subject_id').reset_index(drop=True)
atomic_write_csv(audit_df, POLICY_AUDIT_PATH)

if integrity_errors:
    preview = '\n'.join(integrity_errors[:25])
    raise RuntimeError(
        'Trial8 atlas-policy integrity failed. Do not freeze the policy.\n'
        + preview
    )

if len(audit_df) != EXPECTED_SUBJECTS:
    raise RuntimeError('Atlas-policy audit does not contain 148 subjects.')
if not audit_df['case_integrity_passed'].map(as_bool).all():
    raise RuntimeError('An atlas-policy case failed integrity verification.')

atlas_columns = audit_df[[
    'subject_id',
    'selected_mode',
    'atlas_enabled',
    'registration_confidence',
    'selection_reason',
    'manual_visual_qa_approved',
    'selected_fine_labels_path',
    'selected_vascular30_path',
    'selected_level2_path',
    'selected_atlas_valid_mask_path',
    'selected_metrics_path',
    'selected_metrics_sha256',
    'selected_fine_sha256',
    'selected_vascular30_sha256',
    'selected_level2_sha256',
    'selected_atlas_valid_sha256',
]].copy()
atlas_columns = atlas_columns.rename(columns={
    'selected_mode': 'trial8_atlas_mode',
    'atlas_enabled': 'trial8_atlas_enabled',
    'registration_confidence': 'trial8_registration_confidence',
    'selection_reason': 'trial8_atlas_selection_reason',
    'manual_visual_qa_approved': 'trial8_atlas_manual_qa_approved',
    'selected_fine_labels_path': 'trial8_atlas_fine_labels_path',
    'selected_vascular30_path': 'trial8_atlas_vascular30_path',
    'selected_level2_path': 'trial8_atlas_level2_path',
    'selected_atlas_valid_mask_path': 'trial8_atlas_valid_mask_path',
    'selected_metrics_path': 'trial8_atlas_metrics_path',
    'selected_metrics_sha256': 'trial8_atlas_metrics_sha256',
    'selected_fine_sha256': 'trial8_atlas_fine_labels_sha256',
    'selected_vascular30_sha256': 'trial8_atlas_vascular30_sha256',
    'selected_level2_sha256': 'trial8_atlas_level2_sha256',
    'selected_atlas_valid_sha256': 'trial8_atlas_valid_mask_sha256',
})

frozen_model_manifest_df = (
    dataset_manifest_df.reset_index(drop=True)
    .merge(atlas_columns, on='subject_id', how='left', validate='one_to_one')
)
frozen_model_manifest_df['trial8_atlas_policy_version'] = POLICY_VERSION
frozen_model_manifest_df['trial8_atlas_policy_frozen'] = True
frozen_model_manifest_df['trial8_graph_feature_generation_authorized'] = True
frozen_model_manifest_df['trial8_model_training_authorized'] = False

manifest_checks = {
    'frozen_manifest_has_148_subjects': (
        len(frozen_model_manifest_df) == EXPECTED_SUBJECTS
    ),
    'frozen_manifest_subjects_are_unique': (
        frozen_model_manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'frozen_manifest_split_counts_match': (
        frozen_model_manifest_df['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
    'frozen_manifest_mode_counts_match': (
        frozen_model_manifest_df['trial8_atlas_mode'].value_counts().to_dict()
        == EXPECTED_MODE_COUNTS
    ),
    'frozen_manifest_policy_is_frozen': (
        frozen_model_manifest_df['trial8_atlas_policy_frozen'].map(as_bool).all()
    ),
    'graph_feature_generation_authorized': (
        frozen_model_manifest_df[
            'trial8_graph_feature_generation_authorized'
        ].map(as_bool).all()
    ),
    'model_training_not_authorized': (
        not frozen_model_manifest_df[
            'trial8_model_training_authorized'
        ].map(as_bool).any()
    ),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError(
        'Frozen model-manifest checks failed: ' + ', '.join(failed)
    )

atomic_write_csv(frozen_model_manifest_df, FROZEN_MODEL_MANIFEST_PATH)
frozen_manifest_sha256 = file_sha256(FROZEN_MODEL_MANIFEST_PATH)

policy_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'code_revision': CODE_REVISION,
    'policy_version': POLICY_VERSION,
    'source_registration_version': SOURCE_REGISTRATION_VERSION,
    'fallback_version': FALLBACK_VERSION,
    'mode_counts': EXPECTED_MODE_COUNTS,
    'affine_subjects': AFFINE_SUBJECTS,
    'disabled_subjects': DISABLED_SUBJECTS,
    'manual_affine_qa_approval': MANUAL_AFFINE_QA_APPROVAL,
    'disabled_case_behavior': {
        'subject_retained_for_segmentation': True,
        'atlas_tensors_zeroed': True,
        'atlas_valid_zeroed': True,
        'registration_confidence': 0.0,
        'main_image_pathway_available': True,
    },
    'downstream_requirements': {
        'graph_features_must_respect_atlas_valid': True,
        'model_must_receive_atlas_enabled': True,
        'model_must_receive_registration_confidence': True,
        'training_must_use_random_atlas_dropout': True,
    },
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(policy_config, POLICY_CONFIG_PATH)

freeze_checks = {
    **input_checks,
    'all_148_case_integrity_checks_passed': (
        audit_df['case_integrity_passed'].map(as_bool).all()
    ),
    'audit_mode_counts_match': (
        audit_df['selected_mode'].value_counts().to_dict()
        == EXPECTED_MODE_COUNTS
    ),
    'frozen_model_manifest_checks_passed': all(manifest_checks.values()),
    'frozen_model_manifest_exists': FROZEN_MODEL_MANIFEST_PATH.exists(),
    'policy_config_exists': POLICY_CONFIG_PATH.exists(),
    'target_arrays_not_read': True,
}

freeze_gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'code_revision': CODE_REVISION,
    'policy_version': POLICY_VERSION,
    'policy_frozen': bool(all(freeze_checks.values())),
    'graph_feature_generation_authorized': bool(all(freeze_checks.values())),
    'model_training_authorized': False,
    'mode_counts': EXPECTED_MODE_COUNTS,
    'affine_subjects': AFFINE_SUBJECTS,
    'disabled_subjects': DISABLED_SUBJECTS,
    'manual_affine_qa_approval': MANUAL_AFFINE_QA_APPROVAL,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'freeze_checks': freeze_checks,
    'frozen_dataset_manifest_path': str(FROZEN_DATASET_MANIFEST_PATH),
    'frozen_dataset_manifest_sha256': file_sha256(
        FROZEN_DATASET_MANIFEST_PATH
    ),
    'candidate_selection_path': str(CANDIDATE_SELECTION_PATH),
    'candidate_selection_sha256': file_sha256(CANDIDATE_SELECTION_PATH),
    'fallback_summary_path': str(FALLBACK_SUMMARY_PATH),
    'fallback_summary_sha256': file_sha256(FALLBACK_SUMMARY_PATH),
    'policy_config_path': str(POLICY_CONFIG_PATH),
    'policy_config_sha256': file_sha256(POLICY_CONFIG_PATH),
    'policy_audit_path': str(POLICY_AUDIT_PATH),
    'policy_audit_sha256': file_sha256(POLICY_AUDIT_PATH),
    'frozen_model_manifest_path': str(FROZEN_MODEL_MANIFEST_PATH),
    'frozen_model_manifest_sha256': frozen_manifest_sha256,
}
atomic_write_json(freeze_gate, POLICY_GATE_PATH)

if not freeze_gate['policy_frozen']:
    raise RuntimeError('Trial8 atlas policy did not pass its freeze gate.')

print('\nFinal atlas mode counts by split:')
mode_split_df = audit_df.groupby(
    ['dev_split', 'selected_mode'],
    as_index=False,
).agg(subjects=('subject_id', 'count'))
display(mode_split_df)

print('\nFinal atlas policy summary:')
display(pd.DataFrame([
    {
        'selected_mode': mode,
        'subjects': int((audit_df['selected_mode'] == mode).sum()),
        'atlas_enabled': mode != 'disabled',
    }
    for mode in ['syn', 'affine', 'disabled']
]))

print('\nFreeze checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in freeze_checks.items()
]))

print('\n' + '=' * 80)
print('TRIAL8 ATLAS POLICY FREEZE PASSED')
print('=' * 80)
print(f'Frozen subjects:                     {len(audit_df)}')
print(f'SyN subjects:                        {EXPECTED_MODE_COUNTS["syn"]}')
print(f'Affine subjects:                     {EXPECTED_MODE_COUNTS["affine"]}')
print(f'Atlas-disabled subjects:             {EXPECTED_MODE_COUNTS["disabled"]}')
print(f'Affine visual QA approved:           {AFFINE_SUBJECTS}')
print('Target arrays read:                  False')
print('Study trainable parameters:          0 (policy freeze only)')
print('Atlas policy frozen:                 True')
print('Graph feature generation authorized: True')
print('Model training authorized:           False')
print(f'Frozen model manifest:               {FROZEN_MODEL_MANIFEST_PATH}')
print(f'Frozen manifest SHA-256:             {frozen_manifest_sha256}')
print(f'Policy integrity audit:              {POLICY_AUDIT_PATH}')
print(f'Policy configuration:                {POLICY_CONFIG_PATH}')
print(f'Policy freeze gate:                  {POLICY_GATE_PATH}')
print('\nNext: generate and verify vascular graph-feature sidecars.')


In [ ]:
import sys
import os
import json
import hashlib
import subprocess
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R1_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
SCHEMA_VERSION = 'vascular_hierarchy_v1'
POLICY_VERSION = 'v1_syn139_affine2_disabled7_frozen'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
CODE_REVISION = 'r1_compact_masked_relational_sidecars'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_MODES = {'syn': 139, 'affine': 2, 'disabled': 7}
EXPECTED_MANIFEST_SHA256 = (
    '61b58b16cd28313aa2fbfea6591facb4d504b56241fa9b8789c3c5a755244cd5'
)

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_atlas_policy_frozen.csv'
)

R1_ATLAS_ROOT = PROJECT_ROOT / 'preprocessing' / R1_VERSION / 'vascular_atlas'
SCHEMA_ROOT = R1_ATLAS_ROOT / 'schema' / SCHEMA_VERSION
SCHEMA_CSV_PATH = SCHEMA_ROOT / 'vascular_label_schema.csv'
SCHEMA_GATE_PATH = SCHEMA_ROOT / 'schema_freeze_gate.json'
TEMPLATE_VASCULAR30_PATH = SCHEMA_ROOT / 'ArterialAtlas_fine_vascular_only.nii.gz'

POLICY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_atlas'
    / 'bulk_registration'
    / 'v1_jhu_to_ncct_synra_synthstrip_r2_all148'
    / 'policy_freeze'
    / POLICY_VERSION
)
POLICY_GATE_PATH = POLICY_ROOT / 'atlas_policy_freeze_gate.json'
POLICY_CONFIG_PATH = POLICY_ROOT / 'atlas_policy_config.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
SIDECAR_DIR = OUTPUT_ROOT / 'sidecars'
COMPLETION_DIR = OUTPUT_ROOT / 'completion_records'
GRAPH_SCHEMA_NPZ_PATH = OUTPUT_ROOT / 'graph_schema.npz'
GRAPH_SCHEMA_JSON_PATH = OUTPUT_ROOT / 'graph_schema.json'
AUDIT_CSV_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_audit.csv'
ERROR_CSV_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_errors.csv'
SUMMARY_JSON_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_summary.json'
GENERATION_GATE_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_gate.json'
CANDIDATE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_candidate.csv'
)

NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
MIN_STATS_VOXELS = 16
NODE_RELIABILITY_FULL_SUPPORT_VOXELS = 256

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

CONTINUOUS_CHANNELS = EXPECTED_INPUT_CHANNELS[:13]
STAT_NAMES = ['mean', 'std', 'p10', 'p90']

EXPECTED_VESSEL_NAMES = [
    'BA',
    'R_PCA',
    'L_PCA',
    'R_ICA',
    'R_MCA',
    'L_ICA',
    'L_MCA',
    'R_Pcom',
    'L_Pcom',
    'Acom',
    'R_ACA',
    'L_ACA',
    'third_A2',
]

EXPECTED_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

META_FEATURE_NAMES = [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in CONTINUOUS_CHANNELS
    for stat in STAT_NAMES
] + META_FEATURE_NAMES

CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]

TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}

REQUIRED_NPZ_KEYS = {
    'image',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'subject_id',
}

if TARGET_KEYS & REQUIRED_NPZ_KEYS:
    raise RuntimeError('A target key was accidentally added to REQUIRED_NPZ_KEYS.')


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


for output_path in [
    OUTPUT_ROOT,
    SIDECAR_DIR,
    COMPLETION_DIR,
    CANDIDATE_MANIFEST_PATH.parent,
]:
    assert_project_path_is_safe(output_path)
    output_path.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary_path = Path(str(path) + '.tmp')
    temporary_path.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary_path, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary_path = Path(str(path) + '.tmp')
    frame.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def atomic_write_npz(path, **arrays):
    path = Path(path)
    temporary_path = Path(str(path) + '.tmp')
    with temporary_path.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary_path, path)


def load_effective_3d(path, dtype=None):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    elif array.ndim != 3:
        raise RuntimeError(f'Unsupported 3D image shape: {path} -> {array.shape}')
    if dtype is not None:
        array = array.astype(dtype, copy=False)
    return image, array


def decode_string_array(array):
    return [str(value) for value in np.asarray(array).tolist()]


def normalized_string(value):
    return str(np.asarray(value).item())






required_paths = [
    FROZEN_MANIFEST_PATH,
    SCHEMA_CSV_PATH,
    SCHEMA_GATE_PATH,
    TEMPLATE_VASCULAR30_PATH,
    POLICY_GATE_PATH,
    POLICY_CONFIG_PATH,
]
missing_paths = [str(path) for path in required_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 graph-sidecar prerequisites are missing:\n' + '\n'.join(missing_paths)
    )

observed_manifest_sha256 = file_sha256(FROZEN_MANIFEST_PATH)
if observed_manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        'The frozen atlas-policy manifest changed. '
        f'Observed {observed_manifest_sha256}; expected {EXPECTED_MANIFEST_SHA256}.'
    )

schema_gate = json.loads(SCHEMA_GATE_PATH.read_text(encoding='utf-8'))
policy_gate = json.loads(POLICY_GATE_PATH.read_text(encoding='utf-8'))
policy_config = json.loads(POLICY_CONFIG_PATH.read_text(encoding='utf-8'))

if schema_gate.get('gate_passed') is not True:
    raise RuntimeError('The vascular schema gate is not passed.')
if policy_gate.get('policy_frozen') is not True:
    raise RuntimeError('The final atlas policy is not frozen.')
if policy_gate.get('graph_feature_generation_authorized') is not True:
    raise RuntimeError('The atlas policy does not authorize graph features.')
if policy_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Model training must remain locked during sidecar generation.')
if policy_gate.get('policy_version') != POLICY_VERSION:
    raise RuntimeError('The policy gate version differs from the expected version.')
if policy_gate.get('frozen_model_manifest_sha256') != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('The policy gate points to a different frozen manifest.')

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'cow_prior_valid',
    'trial8_npz_sha256',
    'trial8_synthstrip_mask_path',
    'trial8_synthstrip_mask_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_registration_confidence',
    'trial8_atlas_vascular30_path',
    'trial8_atlas_valid_mask_path',
    'trial8_atlas_vascular30_sha256',
    'trial8_atlas_valid_mask_sha256',
    'trial8_atlas_policy_frozen',
    'trial8_graph_feature_generation_authorized',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen manifest is missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_are_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'atlas_mode_counts_match': manifest_df['trial8_atlas_mode'].value_counts().to_dict()
    == EXPECTED_MODES,
    'atlas_policy_frozen_for_all': manifest_df[
        'trial8_atlas_policy_frozen'
    ].map(as_bool).all(),
    'graph_generation_authorized_for_all': manifest_df[
        'trial8_graph_feature_generation_authorized'
    ].map(as_bool).all(),
    'training_locked_for_all': ~manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).any(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Frozen manifest checks failed: ' + ', '.join(failed))

schema_df = pd.read_csv(SCHEMA_CSV_PATH)
node_schema_df = (
    schema_df[schema_df['use_as_graph_node'].map(as_bool)]
    .sort_values('fine_label')
    .reset_index(drop=True)
)
if node_schema_df['fine_label'].astype(int).tolist() != list(range(1, 31)):
    raise RuntimeError('The frozen graph-node labels are not exactly 1..30.')

print('Trial8 graph-feature gates passed.')
print(f'Frozen subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Atlas modes: {manifest_df["trial8_atlas_mode"].value_counts().to_dict()}')
print(f'Frozen manifest SHA-256: {observed_manifest_sha256}')
print('Target arrays read: False')
print('Study trainable parameters: 0 (feature generation only)')






TERRITORY_RELATION_NAMES = [
    'self',
    'template_contact_26n',
    'contralateral_homolog',
    'same_unilateral_parent',
]

VESSEL_RELATION_NAMES = [
    'self',
    'anatomical_connection',
    'contralateral_homolog',
]

CROSS_RELATION_NAMES = [
    'primary_supply',
    'communicating_or_proximal_supply',
]


def positive_half_offsets_26n():
    offsets = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                if (dx, dy, dz) == (0, 0, 0):
                    continue
                if (dx, dy, dz) > (0, 0, 0):
                    offsets.append((dx, dy, dz))
    return offsets


def shifted_pair(array, offset):
    first_slices = []
    second_slices = []
    for delta in offset:
        if delta > 0:
            first_slices.append(slice(0, -delta))
            second_slices.append(slice(delta, None))
        elif delta < 0:
            first_slices.append(slice(-delta, None))
            second_slices.append(slice(0, delta))
        else:
            first_slices.append(slice(None))
            second_slices.append(slice(None))
    return array[tuple(first_slices)], array[tuple(second_slices)]


def template_contact_counts(label_array):
    counts = {}
    for offset in positive_half_offsets_26n():
        first, second = shifted_pair(label_array, offset)
        valid = (
            (first >= 1)
            & (first <= NUM_TERRITORY_NODES)
            & (second >= 1)
            & (second <= NUM_TERRITORY_NODES)
            & (first != second)
        )
        if not np.any(valid):
            continue
        lower = np.minimum(first[valid], second[valid]).astype(np.int16)
        upper = np.maximum(first[valid], second[valid]).astype(np.int16)
        encoded = lower.astype(np.int32) * 64 + upper.astype(np.int32)
        unique_codes, code_counts = np.unique(encoded, return_counts=True)
        for code, count in zip(unique_codes, code_counts):
            left = int(code // 64)
            right = int(code % 64)
            counts[(left, right)] = counts.get((left, right), 0) + int(count)
    return counts


def build_territory_edges(template_labels, schema):
    edge_rows = []

    def add_bidirectional(label_a, label_b, relation, weight):
        a = int(label_a) - 1
        b = int(label_b) - 1
        edge_rows.append((a, b, relation, float(weight)))
        if a != b:
            edge_rows.append((b, a, relation, float(weight)))

    for label in range(1, NUM_TERRITORY_NODES + 1):
        add_bidirectional(label, label, 0, 1.0)

    contact_counts = template_contact_counts(template_labels)
    maximum_log_contact = max(
        [np.log1p(count) for count in contact_counts.values()],
        default=1.0,
    )
    for (label_a, label_b), count in sorted(contact_counts.items()):
        weight = float(np.log1p(count) / maximum_log_contact)
        add_bidirectional(label_a, label_b, 1, weight)

    for left_label in range(1, NUM_TERRITORY_NODES + 1, 2):
        add_bidirectional(left_label, left_label + 1, 2, 1.0)

    for _, parent_frame in schema.groupby('level2_label', sort=True):
        labels = sorted(parent_frame['fine_label'].astype(int).tolist())
        for label_a, label_b in combinations(labels, 2):
            add_bidirectional(label_a, label_b, 3, 1.0)

    edge_rows = sorted(set(edge_rows))
    edge_index = np.asarray(
        [[row[0] for row in edge_rows], [row[1] for row in edge_rows]],
        dtype=np.int16,
    )
    edge_type = np.asarray([row[2] for row in edge_rows], dtype=np.int8)
    edge_weight = np.asarray([row[3] for row in edge_rows], dtype=np.float32)
    return edge_index, edge_type, edge_weight, contact_counts


def build_vessel_edges(vessel_names):
    index = {name: position for position, name in enumerate(vessel_names)}
    anatomical_pairs = [
        ('BA', 'R_PCA'),
        ('BA', 'L_PCA'),
        ('R_ICA', 'R_MCA'),
        ('R_ICA', 'R_ACA'),
        ('R_ICA', 'R_Pcom'),
        ('L_ICA', 'L_MCA'),
        ('L_ICA', 'L_ACA'),
        ('L_ICA', 'L_Pcom'),
        ('R_Pcom', 'R_PCA'),
        ('L_Pcom', 'L_PCA'),
        ('Acom', 'R_ACA'),
        ('Acom', 'L_ACA'),
        ('third_A2', 'R_ACA'),
        ('third_A2', 'L_ACA'),
    ]
    homolog_pairs = [
        ('R_PCA', 'L_PCA'),
        ('R_ICA', 'L_ICA'),
        ('R_MCA', 'L_MCA'),
        ('R_Pcom', 'L_Pcom'),
        ('R_ACA', 'L_ACA'),
    ]
    rows = []

    def add_pair(name_a, name_b, relation):
        a = index[name_a]
        b = index[name_b]
        rows.append((a, b, relation, 1.0))
        if a != b:
            rows.append((b, a, relation, 1.0))

    for vessel_name in vessel_names:
        add_pair(vessel_name, vessel_name, 0)
    for name_a, name_b in anatomical_pairs:
        add_pair(name_a, name_b, 1)
    for name_a, name_b in homolog_pairs:
        add_pair(name_a, name_b, 2)

    rows = sorted(set(rows))
    edge_index = np.asarray(
        [[row[0] for row in rows], [row[1] for row in rows]], dtype=np.int16
    )
    edge_type = np.asarray([row[2] for row in rows], dtype=np.int8)
    edge_weight = np.asarray([row[3] for row in rows], dtype=np.float32)
    return edge_index, edge_type, edge_weight


def build_vessel_to_territory_edges(vessel_names):
    vessel_index = {name: position for position, name in enumerate(vessel_names)}
    primary_mapping = {
        'BA': [25, 26, 27, 28, 29, 30],
        'R_PCA': [18, 20, 22, 24],
        'L_PCA': [17, 19, 21, 23],
        'R_ICA': [2, 4, 6, 8, 10, 12, 14, 16],
        'L_ICA': [1, 3, 5, 7, 9, 11, 13, 15],
        'R_MCA': [6, 8, 10, 12, 14, 16],
        'L_MCA': [5, 7, 9, 11, 13, 15],
        'R_ACA': [2, 4],
        'L_ACA': [1, 3],
        'third_A2': [1, 2],
    }
    communicating_mapping = {
        'BA': [17, 18, 19, 20, 21, 22, 23, 24],
        'R_Pcom': [18, 20, 22, 24],
        'L_Pcom': [17, 19, 21, 23],
        'Acom': [1, 2, 3, 4],
    }
    rows = []
    for relation, mapping in enumerate([primary_mapping, communicating_mapping]):
        for vessel_name, territory_labels in mapping.items():
            for territory_label in territory_labels:
                rows.append(
                    (
                        vessel_index[vessel_name],
                        int(territory_label) - 1,
                        relation,
                        1.0,
                    )
                )
    rows = sorted(set(rows))
    edge_index = np.asarray(
        [[row[0] for row in rows], [row[1] for row in rows]], dtype=np.int16
    )
    edge_type = np.asarray([row[2] for row in rows], dtype=np.int8)
    edge_weight = np.asarray([row[3] for row in rows], dtype=np.float32)
    return edge_index, edge_type, edge_weight


_, template_labels = load_effective_3d(TEMPLATE_VASCULAR30_PATH, dtype=np.int16)
if set(int(value) for value in np.unique(template_labels)) != set(range(31)):
    raise RuntimeError('The template vascular atlas labels are not exactly 0..30.')

(
    territory_edge_index,
    territory_edge_type,
    territory_edge_weight,
    template_contacts,
) = build_territory_edges(template_labels, node_schema_df)
(
    vessel_edge_index,
    vessel_edge_type,
    vessel_edge_weight,
) = build_vessel_edges(EXPECTED_VESSEL_NAMES)
(
    cross_edge_index,
    cross_edge_type,
    cross_edge_weight,
) = build_vessel_to_territory_edges(EXPECTED_VESSEL_NAMES)

node_labels = node_schema_df['fine_label'].to_numpy(dtype=np.int16)
node_names = node_schema_df['fine_name'].astype(str).to_numpy()
node_acronyms = node_schema_df['fine_acronym'].astype(str).to_numpy()
node_hemisphere = node_schema_df['hemisphere'].astype(str).to_numpy()
node_parent_labels = node_schema_df['level2_label'].to_numpy(dtype=np.int16)
node_parent_acronyms = node_schema_df['level2_acronym'].astype(str).to_numpy()
node_bilateral_family = node_schema_df['bilateral_family'].astype(str).to_numpy()
node_major_group = node_schema_df['major_group'].astype(str).to_numpy()

graph_schema_arrays = {
    'feature_version': np.asarray(FEATURE_VERSION),
    'node_labels': node_labels,
    'node_names': node_names,
    'node_acronyms': node_acronyms,
    'node_hemisphere': node_hemisphere,
    'node_parent_labels': node_parent_labels,
    'node_parent_acronyms': node_parent_acronyms,
    'node_bilateral_family': node_bilateral_family,
    'node_major_group': node_major_group,
    'node_feature_names': np.asarray(NODE_FEATURE_NAMES),
    'case_feature_names': np.asarray(CASE_FEATURE_NAMES),
    'vessel_names': np.asarray(EXPECTED_VESSEL_NAMES),
    'vessel_feature_names': np.asarray(EXPECTED_VESSEL_FEATURE_NAMES),
    'territory_edge_index': territory_edge_index,
    'territory_edge_type': territory_edge_type,
    'territory_edge_weight': territory_edge_weight,
    'territory_relation_names': np.asarray(TERRITORY_RELATION_NAMES),
    'vessel_edge_index': vessel_edge_index,
    'vessel_edge_type': vessel_edge_type,
    'vessel_edge_weight': vessel_edge_weight,
    'vessel_relation_names': np.asarray(VESSEL_RELATION_NAMES),
    'cross_edge_index': cross_edge_index,
    'cross_edge_type': cross_edge_type,
    'cross_edge_weight': cross_edge_weight,
    'cross_relation_names': np.asarray(CROSS_RELATION_NAMES),
}


def graph_schema_file_matches(path, expected_arrays):
    if not Path(path).exists():
        return False
    try:
        with np.load(path, allow_pickle=False) as data:
            if set(data.files) != set(expected_arrays):
                return False
            return all(
                np.array_equal(np.asarray(data[name]), np.asarray(expected))
                for name, expected in expected_arrays.items()
            )
    except Exception:
        return False




if not graph_schema_file_matches(GRAPH_SCHEMA_NPZ_PATH, graph_schema_arrays):
    atomic_write_npz(GRAPH_SCHEMA_NPZ_PATH, **graph_schema_arrays)
graph_schema_sha256 = file_sha256(GRAPH_SCHEMA_NPZ_PATH)

graph_schema_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'feature_version': FEATURE_VERSION,
    'code_revision': CODE_REVISION,
    'schema_version': SCHEMA_VERSION,
    'policy_version': POLICY_VERSION,
    'frozen_manifest_sha256': observed_manifest_sha256,
    'graph_schema_npz_path': str(GRAPH_SCHEMA_NPZ_PATH),
    'graph_schema_npz_sha256': graph_schema_sha256,
    'territory_nodes': NUM_TERRITORY_NODES,
    'territory_features': len(NODE_FEATURE_NAMES),
    'territory_edges': int(territory_edge_index.shape[1]),
    'territory_relation_names': TERRITORY_RELATION_NAMES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'vessel_features': len(EXPECTED_VESSEL_FEATURE_NAMES),
    'vessel_edges': int(vessel_edge_index.shape[1]),
    'vessel_relation_names': VESSEL_RELATION_NAMES,
    'vessel_to_territory_edges': int(cross_edge_index.shape[1]),
    'cross_relation_names': CROSS_RELATION_NAMES,
    'node_feature_names': NODE_FEATURE_NAMES,
    'case_feature_names': CASE_FEATURE_NAMES,
    'normalization_policy': (
        'Sidecars are unstandardized. Fit all learned scalers on train only, '
        'freeze them, and apply unchanged to validation and test.'
    ),
    'disabled_atlas_policy': (
        'Territory tensors and masks are zero; target-independent case and '
        'valid CoW/LVO vessel features remain available.'
    ),
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(graph_schema_record, GRAPH_SCHEMA_JSON_PATH)

print('\nFixed graph schema:')
display(pd.DataFrame([
    {
        'component': 'territory graph',
        'nodes': NUM_TERRITORY_NODES,
        'features': len(NODE_FEATURE_NAMES),
        'edges': int(territory_edge_index.shape[1]),
        'relations': len(TERRITORY_RELATION_NAMES),
    },
    {
        'component': 'vessel graph',
        'nodes': NUM_VESSEL_NODES,
        'features': len(EXPECTED_VESSEL_FEATURE_NAMES),
        'edges': int(vessel_edge_index.shape[1]),
        'relations': len(VESSEL_RELATION_NAMES),
    },
    {
        'component': 'vessel-to-territory',
        'nodes': NUM_VESSEL_NODES + NUM_TERRITORY_NODES,
        'features': 0,
        'edges': int(cross_edge_index.shape[1]),
        'relations': len(CROSS_RELATION_NAMES),
    },
]))






def safe_stats(values):
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]
    if values.size < MIN_STATS_VOXELS:
        return np.zeros(4, dtype=np.float32), False, int(values.size)
    percentiles = np.percentile(values, [10.0, 90.0]).astype(np.float32)
    result = np.asarray(
        [
            np.mean(values, dtype=np.float64),
            np.std(values, dtype=np.float64),
            percentiles[0],
            percentiles[1],
        ],
        dtype=np.float32,
    )
    if not np.isfinite(result).all():
        return np.zeros(4, dtype=np.float32), False, int(values.size)
    return result, True, int(values.size)


def load_subject_inputs(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    brain_mask_path = Path(str(row['trial8_synthstrip_mask_path']))
    required_case_paths = [npz_path, brain_mask_path]

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_valid_path = Path(str(row['trial8_atlas_valid_mask_path']))
        required_case_paths.extend([atlas_path, atlas_valid_path])
    else:
        atlas_path = None
        atlas_valid_path = None

    missing = [str(path) for path in required_case_paths if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing case inputs:\n' + '\n'.join(missing))

    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: NPZ hash changed after dataset freeze.')
    if file_sha256(brain_mask_path) != str(row['trial8_synthstrip_mask_sha256']):
        raise RuntimeError(f'{subject_id}: brain-mask hash changed after freeze.')
    if atlas_enabled:
        if file_sha256(atlas_path) != str(row['trial8_atlas_vascular30_sha256']):
            raise RuntimeError(f'{subject_id}: selected atlas labels changed.')
        if file_sha256(atlas_valid_path) != str(
            row['trial8_atlas_valid_mask_sha256']
        ):
            raise RuntimeError(f'{subject_id}: selected atlas-valid mask changed.')

    with np.load(npz_path, allow_pickle=False) as data:
        missing_keys = sorted(REQUIRED_NPZ_KEYS - set(data.files))
        if missing_keys:
            raise RuntimeError(f'{subject_id}: missing NPZ keys {missing_keys}')

        
        arrays = {
            key: np.asarray(data[key])
            for key in sorted(REQUIRED_NPZ_KEYS)
        }

    stored_subject_id = normalized_string(arrays['subject_id'])
    if stored_subject_id != subject_id:
        raise RuntimeError(
            f'{subject_id}: NPZ subject ID mismatch: {stored_subject_id}'
        )

    input_channels = decode_string_array(arrays['input_channels'])
    vessel_names = decode_string_array(arrays['vessel_names'])
    vessel_feature_names = decode_string_array(arrays['vessel_feature_names'])
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: unexpected input-channel schema.')
    if vessel_names != EXPECTED_VESSEL_NAMES:
        raise RuntimeError(f'{subject_id}: unexpected vessel-name schema.')
    if vessel_feature_names != EXPECTED_VESSEL_FEATURE_NAMES:
        raise RuntimeError(f'{subject_id}: unexpected vessel-feature schema.')

    image = arrays['image'].astype(np.float32, copy=False)
    if image.ndim != 4 or image.shape[0] != len(EXPECTED_INPUT_CHANNELS):
        raise RuntimeError(f'{subject_id}: invalid image shape {image.shape}.')
    spatial_shape = tuple(int(value) for value in image.shape[1:])

    brain_image, brain_mask_array = load_effective_3d(brain_mask_path)
    brain_mask = brain_mask_array > 0.5
    if brain_mask.shape != spatial_shape:
        raise RuntimeError(f'{subject_id}: brain-mask shape mismatch.')
    if not np.allclose(
        brain_image.affine,
        arrays['affine'].astype(np.float64),
        atol=1e-4,
        rtol=1e-6,
    ):
        raise RuntimeError(f'{subject_id}: brain-mask affine mismatch.')

    if atlas_enabled:
        atlas_image, atlas_array = load_effective_3d(atlas_path)
        valid_image, valid_array = load_effective_3d(atlas_valid_path)
        if atlas_array.shape != spatial_shape or valid_array.shape != spatial_shape:
            raise RuntimeError(f'{subject_id}: atlas shape mismatch.')
        for role, image_object in [('atlas', atlas_image), ('valid', valid_image)]:
            if not np.allclose(
                image_object.affine,
                arrays['affine'].astype(np.float64),
                atol=1e-4,
                rtol=1e-6,
            ):
                raise RuntimeError(f'{subject_id}: {role} affine mismatch.')
        if not np.isfinite(atlas_array).all() or not np.isfinite(valid_array).all():
            raise RuntimeError(f'{subject_id}: nonfinite atlas arrays.')
        if np.max(np.abs(atlas_array - np.rint(atlas_array))) > 1e-4:
            raise RuntimeError(f'{subject_id}: nonintegral atlas labels.')
        atlas_labels = np.rint(atlas_array).astype(np.int16)
        atlas_valid = valid_array > 0.5
        if atlas_labels.min() < 0 or atlas_labels.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
        if not np.array_equal(atlas_valid, atlas_labels > 0):
            raise RuntimeError(f'{subject_id}: atlas-valid mask differs from support.')
        if np.any(atlas_valid & ~brain_mask):
            raise RuntimeError(f'{subject_id}: selected atlas extends outside brain mask.')
    else:
        atlas_labels = np.zeros(spatial_shape, dtype=np.int16)
        atlas_valid = np.zeros(spatial_shape, dtype=bool)

    spatial_keys = [
        'crop_mask',
        'perfusion_valid',
        'mtt_valid',
        'mtt_interpolation_support',
        'cow_label',
        'cow_soft_union',
        'lvo_proximity',
    ]
    for key in spatial_keys:
        if arrays[key].shape != spatial_shape:
            raise RuntimeError(
                f'{subject_id}: {key} shape {arrays[key].shape} '
                f'does not match {spatial_shape}.'
            )

    return {
        'subject_id': subject_id,
        'npz_path': npz_path,
        'image': image,
        'brain_mask': brain_mask,
        'atlas_labels': atlas_labels,
        'atlas_valid': atlas_valid,
        'crop_mask': arrays['crop_mask'] > 0,
        'perfusion_valid': arrays['perfusion_valid'] > 0,
        'mtt_valid': arrays['mtt_valid'] > 0,
        'mtt_interpolation_support': arrays[
            'mtt_interpolation_support'
        ].astype(np.float32, copy=False),
        'cow_label': arrays['cow_label'].astype(np.int16, copy=False),
        'cow_soft_union': arrays['cow_soft_union'].astype(np.float32, copy=False),
        'lvo_proximity': arrays['lvo_proximity'].astype(np.float32, copy=False),
        'lvo_present': bool(np.asarray(arrays['lvo_present']).item()),
        'vessel_features_raw': arrays['vessel_features_raw'].astype(
            np.float32, copy=False
        ),
        'vessel_presence': arrays['vessel_presence'].astype(np.uint8, copy=False),
        'affine': arrays['affine'].astype(np.float64, copy=False),
    }


def extract_subject_features(row, arrays):
    subject_id = arrays['subject_id']
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    atlas_mode = str(row['trial8_atlas_mode'])
    registration_confidence = float(row['trial8_registration_confidence'])
    cow_prior_valid = as_bool(row['cow_prior_valid'])

    if atlas_mode not in {'syn', 'affine', 'disabled'}:
        raise RuntimeError(f'{subject_id}: invalid atlas mode {atlas_mode}.')
    if atlas_enabled != (atlas_mode != 'disabled'):
        raise RuntimeError(f'{subject_id}: atlas-enabled flag conflicts with mode.')
    if not np.isfinite(registration_confidence):
        raise RuntimeError(f'{subject_id}: nonfinite registration confidence.')
    if not 0.0 <= registration_confidence <= 1.0:
        raise RuntimeError(f'{subject_id}: registration confidence outside [0, 1].')

    image = arrays['image']
    atlas_labels = arrays['atlas_labels']
    atlas_valid = arrays['atlas_valid']
    brain_mask = arrays['brain_mask']
    perfusion_valid = arrays['perfusion_valid']
    pair_valid = image[14] > 0.5
    mtt_valid = arrays['mtt_valid']
    mtt_support = arrays['mtt_interpolation_support']
    cow_label = arrays['cow_label']
    cow_soft_union = arrays['cow_soft_union']
    lvo_proximity = arrays['lvo_proximity']

    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image contains nonfinite values.')
    for name, value in [
        ('mtt_interpolation_support', mtt_support),
        ('cow_soft_union', cow_soft_union),
        ('lvo_proximity', lvo_proximity),
    ]:
        if not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} contains nonfinite values.')

    voxel_sizes = np.sqrt(np.sum(arrays['affine'][:3, :3] ** 2, axis=0))
    voxel_volume_ml = float(np.prod(voxel_sizes) / 1000.0)
    brain_voxels = int(brain_mask.sum())
    if brain_voxels <= 0:
        raise RuntimeError(f'{subject_id}: empty brain mask.')
    brain_volume_ml = brain_voxels * voxel_volume_ml

    node_features = np.zeros(
        (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES)), dtype=np.float32
    )
    node_feature_valid = np.zeros_like(node_features, dtype=np.uint8)
    node_valid = np.zeros(NUM_TERRITORY_NODES, dtype=np.uint8)
    node_reliability = np.zeros(NUM_TERRITORY_NODES, dtype=np.float32)
    node_voxels = np.zeros(NUM_TERRITORY_NODES, dtype=np.int32)
    channel_sample_counts = np.zeros(
        (NUM_TERRITORY_NODES, len(CONTINUOUS_CHANNELS)), dtype=np.int32
    )

    if atlas_enabled:
        for node_index, territory_label in enumerate(range(1, 31)):
            territory_mask = (atlas_labels == territory_label) & atlas_valid
            territory_voxels = int(territory_mask.sum())
            node_voxels[node_index] = territory_voxels
            if territory_voxels == 0:
                continue

            node_valid[node_index] = 1
            node_reliability[node_index] = np.float32(
                registration_confidence
                * min(
                    1.0,
                    territory_voxels / NODE_RELIABILITY_FULL_SUPPORT_VOXELS,
                )
            )

            feature_offset = 0
            for channel_index, channel_name in enumerate(CONTINUOUS_CHANNELS):
                measurement_mask = territory_mask
                if channel_name in {'cbf', 'cbv', 'mtt', 'tmax'}:
                    measurement_mask = measurement_mask & perfusion_valid
                elif channel_name.startswith('relative_') or channel_name.startswith(
                    'delta_'
                ):
                    measurement_mask = measurement_mask & pair_valid
                if channel_name == 'mtt':
                    measurement_mask = measurement_mask & mtt_valid

                stats, valid, sample_count = safe_stats(
                    image[channel_index][measurement_mask]
                )
                node_features[
                    node_index, feature_offset : feature_offset + len(STAT_NAMES)
                ] = stats
                node_feature_valid[
                    node_index, feature_offset : feature_offset + len(STAT_NAMES)
                ] = int(valid)
                channel_sample_counts[node_index, channel_index] = sample_count
                feature_offset += len(STAT_NAMES)

            meta_values = np.asarray(
                [
                    np.log1p(territory_voxels),
                    np.log1p(territory_voxels * voxel_volume_ml),
                    territory_voxels / brain_voxels,
                    np.mean(perfusion_valid[territory_mask], dtype=np.float64),
                    np.mean(pair_valid[territory_mask], dtype=np.float64),
                    np.mean(mtt_valid[territory_mask], dtype=np.float64),
                    np.mean(mtt_support[territory_mask], dtype=np.float64),
                    np.mean(cow_soft_union[territory_mask], dtype=np.float64)
                    if cow_prior_valid
                    else 0.0,
                    np.max(cow_soft_union[territory_mask])
                    if cow_prior_valid
                    else 0.0,
                    np.mean(cow_label[territory_mask] > 0, dtype=np.float64)
                    if cow_prior_valid
                    else 0.0,
                    np.mean(lvo_proximity[territory_mask], dtype=np.float64),
                    np.max(lvo_proximity[territory_mask]),
                    np.mean(lvo_proximity[territory_mask] >= 0.5, dtype=np.float64),
                ],
                dtype=np.float32,
            )
            meta_valid = np.ones(len(META_FEATURE_NAMES), dtype=np.uint8)
            if not cow_prior_valid:
                for name in [
                    'cow_soft_union_mean',
                    'cow_soft_union_max',
                    'cow_label_overlap_fraction',
                ]:
                    meta_valid[META_FEATURE_NAMES.index(name)] = 0

            node_features[node_index, feature_offset:] = meta_values
            node_feature_valid[node_index, feature_offset:] = meta_valid

    if not atlas_enabled:
        zero_checks = [
            not node_features.any(),
            not node_feature_valid.any(),
            not node_valid.any(),
            not node_reliability.any(),
            not node_voxels.any(),
            not channel_sample_counts.any(),
        ]
        if not all(zero_checks):
            raise RuntimeError(f'{subject_id}: disabled atlas tensors are not zero.')

    vessel_features = arrays['vessel_features_raw'].copy()
    vessel_presence = arrays['vessel_presence'].copy()
    if vessel_features.shape != (
        NUM_VESSEL_NODES,
        len(EXPECTED_VESSEL_FEATURE_NAMES),
    ):
        raise RuntimeError(
            f'{subject_id}: unexpected vessel feature shape {vessel_features.shape}.'
        )
    if vessel_presence.shape != (NUM_VESSEL_NODES,):
        raise RuntimeError(
            f'{subject_id}: unexpected vessel presence shape {vessel_presence.shape}.'
        )
    if not cow_prior_valid:
        vessel_features.fill(0.0)
        vessel_presence.fill(0)
    vessel_feature_valid = np.broadcast_to(
        (vessel_presence > 0)[:, None], vessel_features.shape
    ).astype(np.uint8, copy=True)

    case_features = np.asarray(
        [
            float(atlas_enabled),
            registration_confidence,
            float(atlas_mode == 'syn'),
            float(atlas_mode == 'affine'),
            float(atlas_mode == 'disabled'),
            float(cow_prior_valid),
            float(arrays['lvo_present']),
            float(np.mean(vessel_presence > 0, dtype=np.float64)),
            float(np.mean(perfusion_valid[brain_mask], dtype=np.float64)),
            float(np.mean(pair_valid[brain_mask], dtype=np.float64)),
            float(np.log1p(brain_volume_ml)),
        ],
        dtype=np.float32,
    )

    finite_checks = [
        np.isfinite(node_features).all(),
        np.isfinite(node_reliability).all(),
        np.isfinite(case_features).all(),
        np.isfinite(vessel_features).all(),
    ]
    if not all(finite_checks):
        raise RuntimeError(f'{subject_id}: generated sidecar contains nonfinite values.')

    return {
        'node_features': node_features,
        'node_feature_valid': node_feature_valid,
        'node_valid': node_valid,
        'node_reliability': node_reliability,
        'node_voxels': node_voxels,
        'channel_sample_counts': channel_sample_counts,
        'case_features': case_features,
        'vessel_features': vessel_features.astype(np.float32, copy=False),
        'vessel_feature_valid': vessel_feature_valid,
        'vessel_presence': vessel_presence.astype(np.uint8, copy=False),
        'brain_volume_ml': brain_volume_ml,
        'atlas_valid_voxels': int(atlas_valid.sum()),
        'perfusion_valid_fraction': float(
            np.mean(perfusion_valid[brain_mask], dtype=np.float64)
        ),
        'pair_valid_fraction': float(
            np.mean(pair_valid[brain_mask], dtype=np.float64)
        ),
        'atlas_enabled': atlas_enabled,
        'atlas_mode': atlas_mode,
        'registration_confidence': registration_confidence,
        'cow_prior_valid': cow_prior_valid,
        'lvo_present': arrays['lvo_present'],
    }


def sidecar_arrays(subject_id, row, extracted):
    return {
        'subject_id': np.asarray(subject_id),
        'feature_version': np.asarray(FEATURE_VERSION),
        'code_revision': np.asarray(CODE_REVISION),
        'node_labels': node_labels,
        'node_features': extracted['node_features'],
        'node_feature_valid': extracted['node_feature_valid'],
        'node_valid': extracted['node_valid'],
        'node_reliability': extracted['node_reliability'],
        'node_voxels': extracted['node_voxels'],
        'channel_sample_counts': extracted['channel_sample_counts'],
        'node_feature_names': np.asarray(NODE_FEATURE_NAMES),
        'case_features': extracted['case_features'],
        'case_feature_names': np.asarray(CASE_FEATURE_NAMES),
        'vessel_features': extracted['vessel_features'],
        'vessel_feature_valid': extracted['vessel_feature_valid'],
        'vessel_presence': extracted['vessel_presence'],
        'vessel_names': np.asarray(EXPECTED_VESSEL_NAMES),
        'vessel_feature_names': np.asarray(EXPECTED_VESSEL_FEATURE_NAMES),
        'atlas_enabled': np.asarray(extracted['atlas_enabled'], dtype=np.uint8),
        'atlas_mode': np.asarray(extracted['atlas_mode']),
        'registration_confidence': np.asarray(
            extracted['registration_confidence'], dtype=np.float32
        ),
        'cow_prior_valid': np.asarray(
            extracted['cow_prior_valid'], dtype=np.uint8
        ),
        'lvo_present': np.asarray(extracted['lvo_present'], dtype=np.uint8),
        'brain_volume_ml': np.asarray(extracted['brain_volume_ml'], dtype=np.float32),
        'atlas_valid_voxels': np.asarray(
            extracted['atlas_valid_voxels'], dtype=np.int64
        ),
        'input_npz_sha256': np.asarray(str(row['trial8_npz_sha256'])),
        'selected_atlas_sha256': np.asarray(
            str(row['trial8_atlas_vascular30_sha256'])
            if extracted['atlas_enabled']
            else ''
        ),
        'graph_schema_sha256': np.asarray(graph_schema_sha256),
        'normalization_fitted': np.asarray(False, dtype=np.uint8),
        'target_arrays_read': np.asarray(False, dtype=np.uint8),
        'study_trainable_parameters': np.asarray(0, dtype=np.int64),
    }


def validate_sidecar(path, subject_id, expected_input_hash):
    with np.load(path, allow_pickle=False) as data:
        required = {
            'subject_id',
            'feature_version',
            'node_features',
            'node_feature_valid',
            'node_valid',
            'node_reliability',
            'node_voxels',
            'channel_sample_counts',
            'case_features',
            'vessel_features',
            'vessel_feature_valid',
            'vessel_presence',
            'input_npz_sha256',
            'graph_schema_sha256',
            'target_arrays_read',
        }
        if not required.issubset(data.files):
            return False, 'missing sidecar arrays'
        checks = [
            normalized_string(data['subject_id']) == subject_id,
            normalized_string(data['feature_version']) == FEATURE_VERSION,
            normalized_string(data['input_npz_sha256']) == expected_input_hash,
            normalized_string(data['graph_schema_sha256']) == graph_schema_sha256,
            data['node_features'].shape
            == (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES)),
            data['node_feature_valid'].shape
            == (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES)),
            data['node_valid'].shape == (NUM_TERRITORY_NODES,),
            data['node_reliability'].shape == (NUM_TERRITORY_NODES,),
            data['node_voxels'].shape == (NUM_TERRITORY_NODES,),
            data['channel_sample_counts'].shape
            == (NUM_TERRITORY_NODES, len(CONTINUOUS_CHANNELS)),
            data['case_features'].shape == (len(CASE_FEATURE_NAMES),),
            data['vessel_features'].shape
            == (NUM_VESSEL_NODES, len(EXPECTED_VESSEL_FEATURE_NAMES)),
            data['vessel_feature_valid'].shape
            == (NUM_VESSEL_NODES, len(EXPECTED_VESSEL_FEATURE_NAMES)),
            data['vessel_presence'].shape == (NUM_VESSEL_NODES,),
            np.isfinite(data['node_features']).all(),
            np.isfinite(data['node_reliability']).all(),
            np.isfinite(data['case_features']).all(),
            np.isfinite(data['vessel_features']).all(),
            not bool(np.asarray(data['target_arrays_read']).item()),
        ]
        if not all(checks):
            return False, 'sidecar validation check failed'
    return True, ''






audit_rows = []
error_rows = []
sorted_manifest = manifest_df.sort_values('subject_id').reset_index(drop=True)

for case_index, row in sorted_manifest.iterrows():
    subject_id = str(row['subject_id'])
    sidecar_path = SIDECAR_DIR / f'{subject_id}_{FEATURE_VERSION}.npz'
    completion_path = COMPLETION_DIR / f'{subject_id}_completion.json'
    status = 'generated'

    try:
        resume_valid = False
        if sidecar_path.exists() and completion_path.exists():
            completion = json.loads(completion_path.read_text(encoding='utf-8'))
            completion_checks = [
                completion.get('subject_id') == subject_id,
                completion.get('feature_version') == FEATURE_VERSION,
                completion.get('input_npz_sha256') == str(row['trial8_npz_sha256']),
                completion.get('graph_schema_sha256') == graph_schema_sha256,
                completion.get('sidecar_sha256') == file_sha256(sidecar_path),
                completion.get('target_arrays_read') is False,
            ]
            sidecar_valid, _ = validate_sidecar(
                sidecar_path, subject_id, str(row['trial8_npz_sha256'])
            )
            resume_valid = bool(all(completion_checks) and sidecar_valid)

        if resume_valid:
            status = 'resumed'
            with np.load(sidecar_path, allow_pickle=False) as data:
                atlas_enabled = bool(np.asarray(data['atlas_enabled']).item())
                atlas_mode = normalized_string(data['atlas_mode'])
                cow_prior_valid = bool(np.asarray(data['cow_prior_valid']).item())
                lvo_present = bool(np.asarray(data['lvo_present']).item())
                registration_confidence = float(
                    np.asarray(data['registration_confidence']).item()
                )
                valid_nodes = int(np.asarray(data['node_valid']).sum())
                reliable_nodes = int(
                    (np.asarray(data['node_reliability']) >= 0.5).sum()
                )
                present_vessels = int(np.asarray(data['vessel_presence']).sum())
                brain_volume_ml = float(np.asarray(data['brain_volume_ml']).item())
                atlas_valid_voxels = int(
                    np.asarray(data['atlas_valid_voxels']).item()
                )
                case_values = np.asarray(data['case_features'], dtype=np.float32)
                perfusion_valid_fraction = float(
                    case_values[CASE_FEATURE_NAMES.index('perfusion_valid_fraction')]
                )
                pair_valid_fraction = float(
                    case_values[
                        CASE_FEATURE_NAMES.index(
                            'contralateral_pair_valid_fraction'
                        )
                    ]
                )
        else:
            arrays = load_subject_inputs(row)
            extracted = extract_subject_features(row, arrays)
            atomic_write_npz(
                sidecar_path,
                **sidecar_arrays(subject_id, row, extracted),
            )
            sidecar_valid, validation_error = validate_sidecar(
                sidecar_path, subject_id, str(row['trial8_npz_sha256'])
            )
            if not sidecar_valid:
                raise RuntimeError(validation_error)

            sidecar_hash = file_sha256(sidecar_path)
            completion_record = {
                'created_at_utc': datetime.now(timezone.utc).isoformat(),
                'subject_id': subject_id,
                'dev_split': str(row['dev_split']),
                'feature_version': FEATURE_VERSION,
                'code_revision': CODE_REVISION,
                'input_npz_path': str(row['vascufate_v2_npz_path']),
                'input_npz_sha256': str(row['trial8_npz_sha256']),
                'graph_schema_sha256': graph_schema_sha256,
                'sidecar_path': str(sidecar_path),
                'sidecar_sha256': sidecar_hash,
                'target_arrays_read': False,
                'study_trainable_parameters': 0,
            }
            atomic_write_json(completion_record, completion_path)

            atlas_enabled = extracted['atlas_enabled']
            atlas_mode = extracted['atlas_mode']
            cow_prior_valid = extracted['cow_prior_valid']
            lvo_present = extracted['lvo_present']
            registration_confidence = extracted['registration_confidence']
            valid_nodes = int(extracted['node_valid'].sum())
            reliable_nodes = int((extracted['node_reliability'] >= 0.5).sum())
            present_vessels = int(extracted['vessel_presence'].sum())
            brain_volume_ml = extracted['brain_volume_ml']
            atlas_valid_voxels = extracted['atlas_valid_voxels']
            perfusion_valid_fraction = extracted['perfusion_valid_fraction']
            pair_valid_fraction = extracted['pair_valid_fraction']

        sidecar_hash = file_sha256(sidecar_path)
        audit_rows.append({
            'subject_id': subject_id,
            'dev_split': str(row['dev_split']),
            'status': status,
            'atlas_mode': atlas_mode,
            'atlas_enabled': atlas_enabled,
            'registration_confidence': registration_confidence,
            'cow_prior_valid': cow_prior_valid,
            'lvo_present': lvo_present,
            'valid_territory_nodes': valid_nodes,
            'reliable_territory_nodes_ge_0p5': reliable_nodes,
            'present_vessels': present_vessels,
            'brain_volume_ml': brain_volume_ml,
            'atlas_valid_voxels': atlas_valid_voxels,
            'perfusion_valid_fraction': perfusion_valid_fraction,
            'contralateral_pair_valid_fraction': pair_valid_fraction,
            'node_feature_shape': str(
                (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES))
            ),
            'vessel_feature_shape': str(
                (NUM_VESSEL_NODES, len(EXPECTED_VESSEL_FEATURE_NAMES))
            ),
            'sidecar_path': str(sidecar_path),
            'sidecar_sha256': sidecar_hash,
            'finite': True,
            'target_arrays_read': False,
            'study_trainable_parameters': 0,
        })

        completed = case_index + 1
        if completed == 1 or completed % 10 == 0 or completed == EXPECTED_SUBJECTS:
            print(
                f'Completed {completed}/{EXPECTED_SUBJECTS}: {subject_id}, '
                f'mode={atlas_mode}, nodes={valid_nodes}, status={status}'
            )

    except Exception as error:
        error_rows.append({
            'subject_id': subject_id,
            'dev_split': str(row['dev_split']),
            'error': f'{type(error).__name__}: {error}',
        })
        print(
            f'Failed {case_index + 1}/{EXPECTED_SUBJECTS}: '
            f'{subject_id}: {type(error).__name__}: {error}'
        )

audit_df = pd.DataFrame(audit_rows)
if len(audit_df):
    audit_df = audit_df.sort_values('subject_id').reset_index(drop=True)
atomic_write_csv(audit_df, AUDIT_CSV_PATH)

error_df = pd.DataFrame(error_rows, columns=['subject_id', 'dev_split', 'error'])
atomic_write_csv(error_df, ERROR_CSV_PATH)

all_subjects_generated = (
    len(audit_df) == EXPECTED_SUBJECTS
    and audit_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    and len(error_df) == 0
)
mode_counts_match = (
    all_subjects_generated
    and audit_df['atlas_mode'].value_counts().to_dict() == EXPECTED_MODES
)
disabled_nodes_zero = (
    all_subjects_generated
    and (
        audit_df.loc[
            audit_df['atlas_mode'] == 'disabled', 'valid_territory_nodes'
        ]
        == 0
    ).all()
)
enabled_nodes_present = (
    all_subjects_generated
    and (
        audit_df.loc[
            audit_df['atlas_mode'] != 'disabled', 'valid_territory_nodes'
        ]
        > 0
    ).all()
)
all_finite = all_subjects_generated and audit_df['finite'].map(as_bool).all()
target_free = all_subjects_generated and not audit_df[
    'target_arrays_read'
].map(as_bool).any()

generation_checks = {
    'frozen_manifest_hash_matches': observed_manifest_sha256
    == EXPECTED_MANIFEST_SHA256,
    'graph_schema_written': GRAPH_SCHEMA_NPZ_PATH.exists(),
    'graph_schema_json_written': GRAPH_SCHEMA_JSON_PATH.exists(),
    'all_148_subjects_generated': all_subjects_generated,
    'mode_counts_match_frozen_policy': mode_counts_match,
    'disabled_subject_territory_nodes_are_zero': disabled_nodes_zero,
    'enabled_subjects_have_territory_nodes': enabled_nodes_present,
    'all_sidecars_are_finite': all_finite,
    'target_arrays_not_read': target_free,
    'study_trainable_parameters_are_zero': True,
}
generation_passed = bool(all(generation_checks.values()))

if generation_passed:
    sidecar_lookup = audit_df.set_index('subject_id')
    candidate_manifest_df = manifest_df.copy()
    candidate_manifest_df['trial8_graph_feature_version'] = FEATURE_VERSION
    candidate_manifest_df['trial8_graph_sidecar_path'] = (
        candidate_manifest_df['subject_id'].map(sidecar_lookup['sidecar_path'])
    )
    candidate_manifest_df['trial8_graph_sidecar_sha256'] = (
        candidate_manifest_df['subject_id'].map(sidecar_lookup['sidecar_sha256'])
    )
    candidate_manifest_df['trial8_graph_schema_path'] = str(GRAPH_SCHEMA_NPZ_PATH)
    candidate_manifest_df['trial8_graph_schema_sha256'] = graph_schema_sha256
    candidate_manifest_df['trial8_graph_feature_generation_complete'] = True
    candidate_manifest_df['trial8_graph_feature_verification_passed'] = False
    candidate_manifest_df['trial8_model_training_authorized'] = False
    atomic_write_csv(candidate_manifest_df, CANDIDATE_MANIFEST_PATH)
    candidate_manifest_sha256 = file_sha256(CANDIDATE_MANIFEST_PATH)
else:
    candidate_manifest_sha256 = ''

summary_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'feature_version': FEATURE_VERSION,
    'code_revision': CODE_REVISION,
    'generation_passed': generation_passed,
    'checks': generation_checks,
    'subjects_expected': EXPECTED_SUBJECTS,
    'subjects_generated': int(len(audit_df)),
    'subjects_failed': int(len(error_df)),
    'mode_counts': audit_df['atlas_mode'].value_counts().to_dict()
    if len(audit_df)
    else {},
    'territory_node_count': NUM_TERRITORY_NODES,
    'territory_feature_count': len(NODE_FEATURE_NAMES),
    'vessel_node_count': NUM_VESSEL_NODES,
    'vessel_feature_count': len(EXPECTED_VESSEL_FEATURE_NAMES),
    'graph_schema_path': str(GRAPH_SCHEMA_NPZ_PATH),
    'graph_schema_sha256': graph_schema_sha256,
    'audit_csv_path': str(AUDIT_CSV_PATH),
    'error_csv_path': str(ERROR_CSV_PATH),
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH)
    if generation_passed
    else '',
    'candidate_manifest_sha256': candidate_manifest_sha256,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'model_training_authorized': False,
}
atomic_write_json(summary_record, SUMMARY_JSON_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'feature_version': FEATURE_VERSION,
    'generation_passed': generation_passed,
    'checks': generation_checks,
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'frozen_manifest_sha256': observed_manifest_sha256,
    'graph_schema_path': str(GRAPH_SCHEMA_NPZ_PATH),
    'graph_schema_sha256': graph_schema_sha256,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH)
    if generation_passed
    else '',
    'candidate_manifest_sha256': candidate_manifest_sha256,
    'graph_feature_verification_passed': False,
    'model_training_authorized': False,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(gate_record, GENERATION_GATE_PATH)

print('\nGeneration checks:')
display(
    pd.DataFrame(
        [
            {'check': name, 'passed': passed}
            for name, passed in generation_checks.items()
        ]
    )
)

if len(audit_df):
    print('\nSidecar summary by atlas mode:')
    display(
        audit_df.groupby('atlas_mode', as_index=False).agg(
            subjects=('subject_id', 'count'),
            mean_valid_territory_nodes=('valid_territory_nodes', 'mean'),
            minimum_valid_territory_nodes=('valid_territory_nodes', 'min'),
            mean_present_vessels=('present_vessels', 'mean'),
            mean_registration_confidence=('registration_confidence', 'mean'),
        )
    )

if not generation_passed:
    if len(error_df):
        print('\nGeneration errors:')
        display(error_df)
    failed_checks = [name for name, passed in generation_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 GRAPH SIDECAR GENERATION FAILED. Do not continue to graph '
        'verification or model training. Failed checks: '
        + ', '.join(failed_checks)
    )

print('\n' + '=' * 80)
print('TRIAL8 TARGET-FREE VASCULAR GRAPH SIDECAR GENERATION PASSED')
print('=' * 80)
print(f'Subjects generated:                  {len(audit_df)}')
print(f'Territory nodes per subject:        {NUM_TERRITORY_NODES}')
print(f'Territory features per node:        {len(NODE_FEATURE_NAMES)}')
print(f'Vessel nodes per subject:           {NUM_VESSEL_NODES}')
print(f'Vessel features per node:           {len(EXPECTED_VESSEL_FEATURE_NAMES)}')
print(f'Territory graph edges:              {territory_edge_index.shape[1]}')
print(f'Vessel graph edges:                 {vessel_edge_index.shape[1]}')
print(f'Vessel-to-territory edges:          {cross_edge_index.shape[1]}')
print('Target arrays read:                 False')
print('Study trainable parameters:         0 (feature generation only)')
print('Graph verification passed:          False')
print('Model training authorized:          False')
print(f'Graph schema:                       {GRAPH_SCHEMA_NPZ_PATH}')
print(f'Generation audit:                   {AUDIT_CSV_PATH}')
print(f'Candidate manifest:                 {CANDIDATE_MANIFEST_PATH}')
print(f'Generation gate:                    {GENERATION_GATE_PATH}')
print('\nNext: run the standalone graph-sidecar verification cell.')


In [ ]:
import sys
import os
import json
import hashlib
import subprocess
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R1_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r1'
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
SCHEMA_VERSION = 'vascular_hierarchy_v1'
POLICY_VERSION = 'v1_syn139_affine2_disabled7_frozen'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
CODE_REVISION = 'r2_full_fine_atlas_validity_semantics'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_MODES = {'syn': 139, 'affine': 2, 'disabled': 7}
EXPECTED_MANIFEST_SHA256 = (
    '61b58b16cd28313aa2fbfea6591facb4d504b56241fa9b8789c3c5a755244cd5'
)

FROZEN_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_atlas_policy_frozen.csv'
)

R1_ATLAS_ROOT = PROJECT_ROOT / 'preprocessing' / R1_VERSION / 'vascular_atlas'
SCHEMA_ROOT = R1_ATLAS_ROOT / 'schema' / SCHEMA_VERSION
SCHEMA_CSV_PATH = SCHEMA_ROOT / 'vascular_label_schema.csv'
SCHEMA_GATE_PATH = SCHEMA_ROOT / 'schema_freeze_gate.json'
TEMPLATE_VASCULAR30_PATH = SCHEMA_ROOT / 'ArterialAtlas_fine_vascular_only.nii.gz'

POLICY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_atlas'
    / 'bulk_registration'
    / 'v1_jhu_to_ncct_synra_synthstrip_r2_all148'
    / 'policy_freeze'
    / POLICY_VERSION
)
POLICY_GATE_PATH = POLICY_ROOT / 'atlas_policy_freeze_gate.json'
POLICY_CONFIG_PATH = POLICY_ROOT / 'atlas_policy_config.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
SIDECAR_DIR = OUTPUT_ROOT / 'sidecars'
COMPLETION_DIR = OUTPUT_ROOT / 'completion_records'
GRAPH_SCHEMA_NPZ_PATH = OUTPUT_ROOT / 'graph_schema.npz'
GRAPH_SCHEMA_JSON_PATH = OUTPUT_ROOT / 'graph_schema.json'
AUDIT_CSV_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_audit.csv'
ERROR_CSV_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_errors.csv'
SUMMARY_JSON_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_summary.json'
GENERATION_GATE_PATH = OUTPUT_ROOT / 'graph_sidecar_generation_gate.json'
CANDIDATE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_candidate.csv'
)

NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
MIN_STATS_VOXELS = 16
NODE_RELIABILITY_FULL_SUPPORT_VOXELS = 256

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

CONTINUOUS_CHANNELS = EXPECTED_INPUT_CHANNELS[:13]
STAT_NAMES = ['mean', 'std', 'p10', 'p90']

EXPECTED_VESSEL_NAMES = [
    'BA',
    'R_PCA',
    'L_PCA',
    'R_ICA',
    'R_MCA',
    'L_ICA',
    'L_MCA',
    'R_Pcom',
    'L_Pcom',
    'Acom',
    'R_ACA',
    'L_ACA',
    'third_A2',
]

EXPECTED_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

META_FEATURE_NAMES = [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in CONTINUOUS_CHANNELS
    for stat in STAT_NAMES
] + META_FEATURE_NAMES

CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]

TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}

REQUIRED_NPZ_KEYS = {
    'image',
    'crop_mask',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'input_channels',
    'vessel_names',
    'vessel_feature_names',
    'affine',
    'subject_id',
}

if TARGET_KEYS & REQUIRED_NPZ_KEYS:
    raise RuntimeError('A target key was accidentally added to REQUIRED_NPZ_KEYS.')


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


for output_path in [
    OUTPUT_ROOT,
    SIDECAR_DIR,
    COMPLETION_DIR,
    CANDIDATE_MANIFEST_PATH.parent,
]:
    assert_project_path_is_safe(output_path)
    output_path.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary_path = Path(str(path) + '.tmp')
    temporary_path.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary_path, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary_path = Path(str(path) + '.tmp')
    frame.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def atomic_write_npz(path, **arrays):
    path = Path(path)
    temporary_path = Path(str(path) + '.tmp')
    with temporary_path.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary_path, path)


def load_effective_3d(path, dtype=None):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    elif array.ndim != 3:
        raise RuntimeError(f'Unsupported 3D image shape: {path} -> {array.shape}')
    if dtype is not None:
        array = array.astype(dtype, copy=False)
    return image, array


def decode_string_array(array):
    return [str(value) for value in np.asarray(array).tolist()]


def normalized_string(value):
    return str(np.asarray(value).item())






required_paths = [
    FROZEN_MANIFEST_PATH,
    SCHEMA_CSV_PATH,
    SCHEMA_GATE_PATH,
    TEMPLATE_VASCULAR30_PATH,
    POLICY_GATE_PATH,
    POLICY_CONFIG_PATH,
]
missing_paths = [str(path) for path in required_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 graph-sidecar prerequisites are missing:\n' + '\n'.join(missing_paths)
    )

observed_manifest_sha256 = file_sha256(FROZEN_MANIFEST_PATH)
if observed_manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        'The frozen atlas-policy manifest changed. '
        f'Observed {observed_manifest_sha256}; expected {EXPECTED_MANIFEST_SHA256}.'
    )

schema_gate = json.loads(SCHEMA_GATE_PATH.read_text(encoding='utf-8'))
policy_gate = json.loads(POLICY_GATE_PATH.read_text(encoding='utf-8'))
policy_config = json.loads(POLICY_CONFIG_PATH.read_text(encoding='utf-8'))

if schema_gate.get('gate_passed') is not True:
    raise RuntimeError('The vascular schema gate is not passed.')
if policy_gate.get('policy_frozen') is not True:
    raise RuntimeError('The final atlas policy is not frozen.')
if policy_gate.get('graph_feature_generation_authorized') is not True:
    raise RuntimeError('The atlas policy does not authorize graph features.')
if policy_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Model training must remain locked during sidecar generation.')
if policy_gate.get('policy_version') != POLICY_VERSION:
    raise RuntimeError('The policy gate version differs from the expected version.')
if policy_gate.get('frozen_model_manifest_sha256') != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('The policy gate points to a different frozen manifest.')

manifest_df = pd.read_csv(FROZEN_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'cow_prior_valid',
    'trial8_npz_sha256',
    'trial8_synthstrip_mask_path',
    'trial8_synthstrip_mask_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_registration_confidence',
    'trial8_atlas_fine_labels_path',
    'trial8_atlas_vascular30_path',
    'trial8_atlas_valid_mask_path',
    'trial8_atlas_fine_labels_sha256',
    'trial8_atlas_vascular30_sha256',
    'trial8_atlas_valid_mask_sha256',
    'trial8_atlas_policy_frozen',
    'trial8_graph_feature_generation_authorized',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Frozen manifest is missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_are_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'atlas_mode_counts_match': manifest_df['trial8_atlas_mode'].value_counts().to_dict()
    == EXPECTED_MODES,
    'atlas_policy_frozen_for_all': manifest_df[
        'trial8_atlas_policy_frozen'
    ].map(as_bool).all(),
    'graph_generation_authorized_for_all': manifest_df[
        'trial8_graph_feature_generation_authorized'
    ].map(as_bool).all(),
    'training_locked_for_all': ~manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).any(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Frozen manifest checks failed: ' + ', '.join(failed))

schema_df = pd.read_csv(SCHEMA_CSV_PATH)
node_schema_df = (
    schema_df[schema_df['use_as_graph_node'].map(as_bool)]
    .sort_values('fine_label')
    .reset_index(drop=True)
)
if node_schema_df['fine_label'].astype(int).tolist() != list(range(1, 31)):
    raise RuntimeError('The frozen graph-node labels are not exactly 1..30.')

print('Trial8 graph-feature gates passed.')
print(f'Frozen subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Atlas modes: {manifest_df["trial8_atlas_mode"].value_counts().to_dict()}')
print(f'Frozen manifest SHA-256: {observed_manifest_sha256}')
print('Target arrays read: False')
print('Study trainable parameters: 0 (feature generation only)')






TERRITORY_RELATION_NAMES = [
    'self',
    'template_contact_26n',
    'contralateral_homolog',
    'same_unilateral_parent',
]

VESSEL_RELATION_NAMES = [
    'self',
    'anatomical_connection',
    'contralateral_homolog',
]

CROSS_RELATION_NAMES = [
    'primary_supply',
    'communicating_or_proximal_supply',
]


def positive_half_offsets_26n():
    offsets = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                if (dx, dy, dz) == (0, 0, 0):
                    continue
                if (dx, dy, dz) > (0, 0, 0):
                    offsets.append((dx, dy, dz))
    return offsets


def shifted_pair(array, offset):
    first_slices = []
    second_slices = []
    for delta in offset:
        if delta > 0:
            first_slices.append(slice(0, -delta))
            second_slices.append(slice(delta, None))
        elif delta < 0:
            first_slices.append(slice(-delta, None))
            second_slices.append(slice(0, delta))
        else:
            first_slices.append(slice(None))
            second_slices.append(slice(None))
    return array[tuple(first_slices)], array[tuple(second_slices)]


def template_contact_counts(label_array):
    counts = {}
    for offset in positive_half_offsets_26n():
        first, second = shifted_pair(label_array, offset)
        valid = (
            (first >= 1)
            & (first <= NUM_TERRITORY_NODES)
            & (second >= 1)
            & (second <= NUM_TERRITORY_NODES)
            & (first != second)
        )
        if not np.any(valid):
            continue
        lower = np.minimum(first[valid], second[valid]).astype(np.int16)
        upper = np.maximum(first[valid], second[valid]).astype(np.int16)
        encoded = lower.astype(np.int32) * 64 + upper.astype(np.int32)
        unique_codes, code_counts = np.unique(encoded, return_counts=True)
        for code, count in zip(unique_codes, code_counts):
            left = int(code // 64)
            right = int(code % 64)
            counts[(left, right)] = counts.get((left, right), 0) + int(count)
    return counts


def build_territory_edges(template_labels, schema):
    edge_rows = []

    def add_bidirectional(label_a, label_b, relation, weight):
        a = int(label_a) - 1
        b = int(label_b) - 1
        edge_rows.append((a, b, relation, float(weight)))
        if a != b:
            edge_rows.append((b, a, relation, float(weight)))

    for label in range(1, NUM_TERRITORY_NODES + 1):
        add_bidirectional(label, label, 0, 1.0)

    contact_counts = template_contact_counts(template_labels)
    maximum_log_contact = max(
        [np.log1p(count) for count in contact_counts.values()],
        default=1.0,
    )
    for (label_a, label_b), count in sorted(contact_counts.items()):
        weight = float(np.log1p(count) / maximum_log_contact)
        add_bidirectional(label_a, label_b, 1, weight)

    for left_label in range(1, NUM_TERRITORY_NODES + 1, 2):
        add_bidirectional(left_label, left_label + 1, 2, 1.0)

    for _, parent_frame in schema.groupby('level2_label', sort=True):
        labels = sorted(parent_frame['fine_label'].astype(int).tolist())
        for label_a, label_b in combinations(labels, 2):
            add_bidirectional(label_a, label_b, 3, 1.0)

    edge_rows = sorted(set(edge_rows))
    edge_index = np.asarray(
        [[row[0] for row in edge_rows], [row[1] for row in edge_rows]],
        dtype=np.int16,
    )
    edge_type = np.asarray([row[2] for row in edge_rows], dtype=np.int8)
    edge_weight = np.asarray([row[3] for row in edge_rows], dtype=np.float32)
    return edge_index, edge_type, edge_weight, contact_counts


def build_vessel_edges(vessel_names):
    index = {name: position for position, name in enumerate(vessel_names)}
    anatomical_pairs = [
        ('BA', 'R_PCA'),
        ('BA', 'L_PCA'),
        ('R_ICA', 'R_MCA'),
        ('R_ICA', 'R_ACA'),
        ('R_ICA', 'R_Pcom'),
        ('L_ICA', 'L_MCA'),
        ('L_ICA', 'L_ACA'),
        ('L_ICA', 'L_Pcom'),
        ('R_Pcom', 'R_PCA'),
        ('L_Pcom', 'L_PCA'),
        ('Acom', 'R_ACA'),
        ('Acom', 'L_ACA'),
        ('third_A2', 'R_ACA'),
        ('third_A2', 'L_ACA'),
    ]
    homolog_pairs = [
        ('R_PCA', 'L_PCA'),
        ('R_ICA', 'L_ICA'),
        ('R_MCA', 'L_MCA'),
        ('R_Pcom', 'L_Pcom'),
        ('R_ACA', 'L_ACA'),
    ]
    rows = []

    def add_pair(name_a, name_b, relation):
        a = index[name_a]
        b = index[name_b]
        rows.append((a, b, relation, 1.0))
        if a != b:
            rows.append((b, a, relation, 1.0))

    for vessel_name in vessel_names:
        add_pair(vessel_name, vessel_name, 0)
    for name_a, name_b in anatomical_pairs:
        add_pair(name_a, name_b, 1)
    for name_a, name_b in homolog_pairs:
        add_pair(name_a, name_b, 2)

    rows = sorted(set(rows))
    edge_index = np.asarray(
        [[row[0] for row in rows], [row[1] for row in rows]], dtype=np.int16
    )
    edge_type = np.asarray([row[2] for row in rows], dtype=np.int8)
    edge_weight = np.asarray([row[3] for row in rows], dtype=np.float32)
    return edge_index, edge_type, edge_weight


def build_vessel_to_territory_edges(vessel_names):
    vessel_index = {name: position for position, name in enumerate(vessel_names)}
    primary_mapping = {
        'BA': [25, 26, 27, 28, 29, 30],
        'R_PCA': [18, 20, 22, 24],
        'L_PCA': [17, 19, 21, 23],
        'R_ICA': [2, 4, 6, 8, 10, 12, 14, 16],
        'L_ICA': [1, 3, 5, 7, 9, 11, 13, 15],
        'R_MCA': [6, 8, 10, 12, 14, 16],
        'L_MCA': [5, 7, 9, 11, 13, 15],
        'R_ACA': [2, 4],
        'L_ACA': [1, 3],
        'third_A2': [1, 2],
    }
    communicating_mapping = {
        'BA': [17, 18, 19, 20, 21, 22, 23, 24],
        'R_Pcom': [18, 20, 22, 24],
        'L_Pcom': [17, 19, 21, 23],
        'Acom': [1, 2, 3, 4],
    }
    rows = []
    for relation, mapping in enumerate([primary_mapping, communicating_mapping]):
        for vessel_name, territory_labels in mapping.items():
            for territory_label in territory_labels:
                rows.append(
                    (
                        vessel_index[vessel_name],
                        int(territory_label) - 1,
                        relation,
                        1.0,
                    )
                )
    rows = sorted(set(rows))
    edge_index = np.asarray(
        [[row[0] for row in rows], [row[1] for row in rows]], dtype=np.int16
    )
    edge_type = np.asarray([row[2] for row in rows], dtype=np.int8)
    edge_weight = np.asarray([row[3] for row in rows], dtype=np.float32)
    return edge_index, edge_type, edge_weight


_, template_labels = load_effective_3d(TEMPLATE_VASCULAR30_PATH, dtype=np.int16)
if set(int(value) for value in np.unique(template_labels)) != set(range(31)):
    raise RuntimeError('The template vascular atlas labels are not exactly 0..30.')

(
    territory_edge_index,
    territory_edge_type,
    territory_edge_weight,
    template_contacts,
) = build_territory_edges(template_labels, node_schema_df)
(
    vessel_edge_index,
    vessel_edge_type,
    vessel_edge_weight,
) = build_vessel_edges(EXPECTED_VESSEL_NAMES)
(
    cross_edge_index,
    cross_edge_type,
    cross_edge_weight,
) = build_vessel_to_territory_edges(EXPECTED_VESSEL_NAMES)

node_labels = node_schema_df['fine_label'].to_numpy(dtype=np.int16)
node_names = node_schema_df['fine_name'].astype(str).to_numpy()
node_acronyms = node_schema_df['fine_acronym'].astype(str).to_numpy()
node_hemisphere = node_schema_df['hemisphere'].astype(str).to_numpy()
node_parent_labels = node_schema_df['level2_label'].to_numpy(dtype=np.int16)
node_parent_acronyms = node_schema_df['level2_acronym'].astype(str).to_numpy()
node_bilateral_family = node_schema_df['bilateral_family'].astype(str).to_numpy()
node_major_group = node_schema_df['major_group'].astype(str).to_numpy()

graph_schema_arrays = {
    'feature_version': np.asarray(FEATURE_VERSION),
    'node_labels': node_labels,
    'node_names': node_names,
    'node_acronyms': node_acronyms,
    'node_hemisphere': node_hemisphere,
    'node_parent_labels': node_parent_labels,
    'node_parent_acronyms': node_parent_acronyms,
    'node_bilateral_family': node_bilateral_family,
    'node_major_group': node_major_group,
    'node_feature_names': np.asarray(NODE_FEATURE_NAMES),
    'case_feature_names': np.asarray(CASE_FEATURE_NAMES),
    'vessel_names': np.asarray(EXPECTED_VESSEL_NAMES),
    'vessel_feature_names': np.asarray(EXPECTED_VESSEL_FEATURE_NAMES),
    'territory_edge_index': territory_edge_index,
    'territory_edge_type': territory_edge_type,
    'territory_edge_weight': territory_edge_weight,
    'territory_relation_names': np.asarray(TERRITORY_RELATION_NAMES),
    'vessel_edge_index': vessel_edge_index,
    'vessel_edge_type': vessel_edge_type,
    'vessel_edge_weight': vessel_edge_weight,
    'vessel_relation_names': np.asarray(VESSEL_RELATION_NAMES),
    'cross_edge_index': cross_edge_index,
    'cross_edge_type': cross_edge_type,
    'cross_edge_weight': cross_edge_weight,
    'cross_relation_names': np.asarray(CROSS_RELATION_NAMES),
}


def graph_schema_file_matches(path, expected_arrays):
    if not Path(path).exists():
        return False
    try:
        with np.load(path, allow_pickle=False) as data:
            if set(data.files) != set(expected_arrays):
                return False
            return all(
                np.array_equal(np.asarray(data[name]), np.asarray(expected))
                for name, expected in expected_arrays.items()
            )
    except Exception:
        return False




if not graph_schema_file_matches(GRAPH_SCHEMA_NPZ_PATH, graph_schema_arrays):
    atomic_write_npz(GRAPH_SCHEMA_NPZ_PATH, **graph_schema_arrays)
graph_schema_sha256 = file_sha256(GRAPH_SCHEMA_NPZ_PATH)

graph_schema_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'feature_version': FEATURE_VERSION,
    'code_revision': CODE_REVISION,
    'schema_version': SCHEMA_VERSION,
    'policy_version': POLICY_VERSION,
    'frozen_manifest_sha256': observed_manifest_sha256,
    'graph_schema_npz_path': str(GRAPH_SCHEMA_NPZ_PATH),
    'graph_schema_npz_sha256': graph_schema_sha256,
    'territory_nodes': NUM_TERRITORY_NODES,
    'territory_features': len(NODE_FEATURE_NAMES),
    'territory_edges': int(territory_edge_index.shape[1]),
    'territory_relation_names': TERRITORY_RELATION_NAMES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'vessel_features': len(EXPECTED_VESSEL_FEATURE_NAMES),
    'vessel_edges': int(vessel_edge_index.shape[1]),
    'vessel_relation_names': VESSEL_RELATION_NAMES,
    'vessel_to_territory_edges': int(cross_edge_index.shape[1]),
    'cross_relation_names': CROSS_RELATION_NAMES,
    'node_feature_names': NODE_FEATURE_NAMES,
    'case_feature_names': CASE_FEATURE_NAMES,
    'normalization_policy': (
        'Sidecars are unstandardized. Fit all learned scalers on train only, '
        'freeze them, and apply unchanged to validation and test.'
    ),
    'disabled_atlas_policy': (
        'Territory tensors and masks are zero; target-independent case and '
        'valid CoW/LVO vessel features remain available.'
    ),
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(graph_schema_record, GRAPH_SCHEMA_JSON_PATH)

print('\nFixed graph schema:')
display(pd.DataFrame([
    {
        'component': 'territory graph',
        'nodes': NUM_TERRITORY_NODES,
        'features': len(NODE_FEATURE_NAMES),
        'edges': int(territory_edge_index.shape[1]),
        'relations': len(TERRITORY_RELATION_NAMES),
    },
    {
        'component': 'vessel graph',
        'nodes': NUM_VESSEL_NODES,
        'features': len(EXPECTED_VESSEL_FEATURE_NAMES),
        'edges': int(vessel_edge_index.shape[1]),
        'relations': len(VESSEL_RELATION_NAMES),
    },
    {
        'component': 'vessel-to-territory',
        'nodes': NUM_VESSEL_NODES + NUM_TERRITORY_NODES,
        'features': 0,
        'edges': int(cross_edge_index.shape[1]),
        'relations': len(CROSS_RELATION_NAMES),
    },
]))






def safe_stats(values):
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]
    if values.size < MIN_STATS_VOXELS:
        return np.zeros(4, dtype=np.float32), False, int(values.size)
    percentiles = np.percentile(values, [10.0, 90.0]).astype(np.float32)
    result = np.asarray(
        [
            np.mean(values, dtype=np.float64),
            np.std(values, dtype=np.float64),
            percentiles[0],
            percentiles[1],
        ],
        dtype=np.float32,
    )
    if not np.isfinite(result).all():
        return np.zeros(4, dtype=np.float32), False, int(values.size)
    return result, True, int(values.size)


def load_subject_inputs(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    brain_mask_path = Path(str(row['trial8_synthstrip_mask_path']))
    required_case_paths = [npz_path, brain_mask_path]

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        fine_atlas_path = Path(str(row['trial8_atlas_fine_labels_path']))
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_valid_path = Path(str(row['trial8_atlas_valid_mask_path']))
        required_case_paths.extend(
            [fine_atlas_path, atlas_path, atlas_valid_path]
        )
    else:
        fine_atlas_path = None
        atlas_path = None
        atlas_valid_path = None

    missing = [str(path) for path in required_case_paths if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing case inputs:\n' + '\n'.join(missing))

    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: NPZ hash changed after dataset freeze.')
    if file_sha256(brain_mask_path) != str(row['trial8_synthstrip_mask_sha256']):
        raise RuntimeError(f'{subject_id}: brain-mask hash changed after freeze.')
    if atlas_enabled:
        if file_sha256(fine_atlas_path) != str(
            row['trial8_atlas_fine_labels_sha256']
        ):
            raise RuntimeError(f'{subject_id}: selected full fine atlas changed.')
        if file_sha256(atlas_path) != str(row['trial8_atlas_vascular30_sha256']):
            raise RuntimeError(f'{subject_id}: selected atlas labels changed.')
        if file_sha256(atlas_valid_path) != str(
            row['trial8_atlas_valid_mask_sha256']
        ):
            raise RuntimeError(f'{subject_id}: selected atlas-valid mask changed.')

    with np.load(npz_path, allow_pickle=False) as data:
        missing_keys = sorted(REQUIRED_NPZ_KEYS - set(data.files))
        if missing_keys:
            raise RuntimeError(f'{subject_id}: missing NPZ keys {missing_keys}')

        
        arrays = {
            key: np.asarray(data[key])
            for key in sorted(REQUIRED_NPZ_KEYS)
        }

    stored_subject_id = normalized_string(arrays['subject_id'])
    if stored_subject_id != subject_id:
        raise RuntimeError(
            f'{subject_id}: NPZ subject ID mismatch: {stored_subject_id}'
        )

    input_channels = decode_string_array(arrays['input_channels'])
    vessel_names = decode_string_array(arrays['vessel_names'])
    vessel_feature_names = decode_string_array(arrays['vessel_feature_names'])
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: unexpected input-channel schema.')
    if vessel_names != EXPECTED_VESSEL_NAMES:
        raise RuntimeError(f'{subject_id}: unexpected vessel-name schema.')
    if vessel_feature_names != EXPECTED_VESSEL_FEATURE_NAMES:
        raise RuntimeError(f'{subject_id}: unexpected vessel-feature schema.')

    image = arrays['image'].astype(np.float32, copy=False)
    if image.ndim != 4 or image.shape[0] != len(EXPECTED_INPUT_CHANNELS):
        raise RuntimeError(f'{subject_id}: invalid image shape {image.shape}.')
    spatial_shape = tuple(int(value) for value in image.shape[1:])

    brain_image, brain_mask_array = load_effective_3d(brain_mask_path)
    brain_mask = brain_mask_array > 0.5
    if brain_mask.shape != spatial_shape:
        raise RuntimeError(f'{subject_id}: brain-mask shape mismatch.')
    if not np.allclose(
        brain_image.affine,
        arrays['affine'].astype(np.float64),
        atol=1e-4,
        rtol=1e-6,
    ):
        raise RuntimeError(f'{subject_id}: brain-mask affine mismatch.')

    if atlas_enabled:
        fine_atlas_image, fine_atlas_array = load_effective_3d(fine_atlas_path)
        atlas_image, atlas_array = load_effective_3d(atlas_path)
        valid_image, valid_array = load_effective_3d(atlas_valid_path)
        if (
            fine_atlas_array.shape != spatial_shape
            or atlas_array.shape != spatial_shape
            or valid_array.shape != spatial_shape
        ):
            raise RuntimeError(f'{subject_id}: atlas shape mismatch.')
        for role, image_object in [
            ('full fine atlas', fine_atlas_image),
            ('vascular30 atlas', atlas_image),
            ('valid', valid_image),
        ]:
            if not np.allclose(
                image_object.affine,
                arrays['affine'].astype(np.float64),
                atol=1e-4,
                rtol=1e-6,
            ):
                raise RuntimeError(f'{subject_id}: {role} affine mismatch.')
        if (
            not np.isfinite(fine_atlas_array).all()
            or not np.isfinite(atlas_array).all()
            or not np.isfinite(valid_array).all()
        ):
            raise RuntimeError(f'{subject_id}: nonfinite atlas arrays.')
        if np.max(np.abs(fine_atlas_array - np.rint(fine_atlas_array))) > 1e-4:
            raise RuntimeError(f'{subject_id}: nonintegral full fine atlas labels.')
        if np.max(np.abs(atlas_array - np.rint(atlas_array))) > 1e-4:
            raise RuntimeError(f'{subject_id}: nonintegral vascular30 labels.')
        fine_atlas_labels = np.rint(fine_atlas_array).astype(np.int16)
        atlas_labels = np.rint(atlas_array).astype(np.int16)
        full_atlas_valid = valid_array > 0.5
        if fine_atlas_labels.min() < 0 or fine_atlas_labels.max() > 32:
            raise RuntimeError(f'{subject_id}: full fine atlas outside 0..32.')
        if atlas_labels.min() < 0 or atlas_labels.max() > 30:
            raise RuntimeError(f'{subject_id}: vascular30 labels outside 0..30.')

        
        
        
        
        if not np.array_equal(full_atlas_valid, fine_atlas_labels > 0):
            raise RuntimeError(
                f'{subject_id}: full atlas-valid mask differs from fine support.'
            )
        expected_vascular30 = fine_atlas_labels.copy()
        expected_vascular30[np.isin(expected_vascular30, [31, 32])] = 0
        if not np.array_equal(atlas_labels, expected_vascular30):
            raise RuntimeError(
                f'{subject_id}: vascular30 map is not the fine atlas with '
                'ventricles removed.'
            )

        atlas_valid = full_atlas_valid & (atlas_labels > 0)
        if not np.array_equal(atlas_valid, atlas_labels > 0):
            raise RuntimeError(
                f'{subject_id}: vascular labels extend outside full valid support.'
            )
        if np.any(atlas_valid & ~brain_mask):
            raise RuntimeError(f'{subject_id}: selected atlas extends outside brain mask.')
    else:
        atlas_labels = np.zeros(spatial_shape, dtype=np.int16)
        atlas_valid = np.zeros(spatial_shape, dtype=bool)

    spatial_keys = [
        'crop_mask',
        'perfusion_valid',
        'mtt_valid',
        'mtt_interpolation_support',
        'cow_label',
        'cow_soft_union',
        'lvo_proximity',
    ]
    for key in spatial_keys:
        if arrays[key].shape != spatial_shape:
            raise RuntimeError(
                f'{subject_id}: {key} shape {arrays[key].shape} '
                f'does not match {spatial_shape}.'
            )

    return {
        'subject_id': subject_id,
        'npz_path': npz_path,
        'image': image,
        'brain_mask': brain_mask,
        'atlas_labels': atlas_labels,
        'atlas_valid': atlas_valid,
        'crop_mask': arrays['crop_mask'] > 0,
        'perfusion_valid': arrays['perfusion_valid'] > 0,
        'mtt_valid': arrays['mtt_valid'] > 0,
        'mtt_interpolation_support': arrays[
            'mtt_interpolation_support'
        ].astype(np.float32, copy=False),
        'cow_label': arrays['cow_label'].astype(np.int16, copy=False),
        'cow_soft_union': arrays['cow_soft_union'].astype(np.float32, copy=False),
        'lvo_proximity': arrays['lvo_proximity'].astype(np.float32, copy=False),
        'lvo_present': bool(np.asarray(arrays['lvo_present']).item()),
        'vessel_features_raw': arrays['vessel_features_raw'].astype(
            np.float32, copy=False
        ),
        'vessel_presence': arrays['vessel_presence'].astype(np.uint8, copy=False),
        'affine': arrays['affine'].astype(np.float64, copy=False),
    }


def extract_subject_features(row, arrays):
    subject_id = arrays['subject_id']
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    atlas_mode = str(row['trial8_atlas_mode'])
    registration_confidence = float(row['trial8_registration_confidence'])
    cow_prior_valid = as_bool(row['cow_prior_valid'])

    if atlas_mode not in {'syn', 'affine', 'disabled'}:
        raise RuntimeError(f'{subject_id}: invalid atlas mode {atlas_mode}.')
    if atlas_enabled != (atlas_mode != 'disabled'):
        raise RuntimeError(f'{subject_id}: atlas-enabled flag conflicts with mode.')
    if not np.isfinite(registration_confidence):
        raise RuntimeError(f'{subject_id}: nonfinite registration confidence.')
    if not 0.0 <= registration_confidence <= 1.0:
        raise RuntimeError(f'{subject_id}: registration confidence outside [0, 1].')

    image = arrays['image']
    atlas_labels = arrays['atlas_labels']
    atlas_valid = arrays['atlas_valid']
    brain_mask = arrays['brain_mask']
    perfusion_valid = arrays['perfusion_valid']
    pair_valid = image[14] > 0.5
    mtt_valid = arrays['mtt_valid']
    mtt_support = arrays['mtt_interpolation_support']
    cow_label = arrays['cow_label']
    cow_soft_union = arrays['cow_soft_union']
    lvo_proximity = arrays['lvo_proximity']

    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image contains nonfinite values.')
    for name, value in [
        ('mtt_interpolation_support', mtt_support),
        ('cow_soft_union', cow_soft_union),
        ('lvo_proximity', lvo_proximity),
    ]:
        if not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} contains nonfinite values.')

    voxel_sizes = np.sqrt(np.sum(arrays['affine'][:3, :3] ** 2, axis=0))
    voxel_volume_ml = float(np.prod(voxel_sizes) / 1000.0)
    brain_voxels = int(brain_mask.sum())
    if brain_voxels <= 0:
        raise RuntimeError(f'{subject_id}: empty brain mask.')
    brain_volume_ml = brain_voxels * voxel_volume_ml

    node_features = np.zeros(
        (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES)), dtype=np.float32
    )
    node_feature_valid = np.zeros_like(node_features, dtype=np.uint8)
    node_valid = np.zeros(NUM_TERRITORY_NODES, dtype=np.uint8)
    node_reliability = np.zeros(NUM_TERRITORY_NODES, dtype=np.float32)
    node_voxels = np.zeros(NUM_TERRITORY_NODES, dtype=np.int32)
    channel_sample_counts = np.zeros(
        (NUM_TERRITORY_NODES, len(CONTINUOUS_CHANNELS)), dtype=np.int32
    )

    if atlas_enabled:
        for node_index, territory_label in enumerate(range(1, 31)):
            territory_mask = (atlas_labels == territory_label) & atlas_valid
            territory_voxels = int(territory_mask.sum())
            node_voxels[node_index] = territory_voxels
            if territory_voxels == 0:
                continue

            node_valid[node_index] = 1
            node_reliability[node_index] = np.float32(
                registration_confidence
                * min(
                    1.0,
                    territory_voxels / NODE_RELIABILITY_FULL_SUPPORT_VOXELS,
                )
            )

            feature_offset = 0
            for channel_index, channel_name in enumerate(CONTINUOUS_CHANNELS):
                measurement_mask = territory_mask
                if channel_name in {'cbf', 'cbv', 'mtt', 'tmax'}:
                    measurement_mask = measurement_mask & perfusion_valid
                elif channel_name.startswith('relative_') or channel_name.startswith(
                    'delta_'
                ):
                    measurement_mask = measurement_mask & pair_valid
                if channel_name == 'mtt':
                    measurement_mask = measurement_mask & mtt_valid

                stats, valid, sample_count = safe_stats(
                    image[channel_index][measurement_mask]
                )
                node_features[
                    node_index, feature_offset : feature_offset + len(STAT_NAMES)
                ] = stats
                node_feature_valid[
                    node_index, feature_offset : feature_offset + len(STAT_NAMES)
                ] = int(valid)
                channel_sample_counts[node_index, channel_index] = sample_count
                feature_offset += len(STAT_NAMES)

            meta_values = np.asarray(
                [
                    np.log1p(territory_voxels),
                    np.log1p(territory_voxels * voxel_volume_ml),
                    territory_voxels / brain_voxels,
                    np.mean(perfusion_valid[territory_mask], dtype=np.float64),
                    np.mean(pair_valid[territory_mask], dtype=np.float64),
                    np.mean(mtt_valid[territory_mask], dtype=np.float64),
                    np.mean(mtt_support[territory_mask], dtype=np.float64),
                    np.mean(cow_soft_union[territory_mask], dtype=np.float64)
                    if cow_prior_valid
                    else 0.0,
                    np.max(cow_soft_union[territory_mask])
                    if cow_prior_valid
                    else 0.0,
                    np.mean(cow_label[territory_mask] > 0, dtype=np.float64)
                    if cow_prior_valid
                    else 0.0,
                    np.mean(lvo_proximity[territory_mask], dtype=np.float64),
                    np.max(lvo_proximity[territory_mask]),
                    np.mean(lvo_proximity[territory_mask] >= 0.5, dtype=np.float64),
                ],
                dtype=np.float32,
            )
            meta_valid = np.ones(len(META_FEATURE_NAMES), dtype=np.uint8)
            if not cow_prior_valid:
                for name in [
                    'cow_soft_union_mean',
                    'cow_soft_union_max',
                    'cow_label_overlap_fraction',
                ]:
                    meta_valid[META_FEATURE_NAMES.index(name)] = 0

            node_features[node_index, feature_offset:] = meta_values
            node_feature_valid[node_index, feature_offset:] = meta_valid

    if not atlas_enabled:
        zero_checks = [
            not node_features.any(),
            not node_feature_valid.any(),
            not node_valid.any(),
            not node_reliability.any(),
            not node_voxels.any(),
            not channel_sample_counts.any(),
        ]
        if not all(zero_checks):
            raise RuntimeError(f'{subject_id}: disabled atlas tensors are not zero.')

    vessel_features = arrays['vessel_features_raw'].copy()
    vessel_presence = arrays['vessel_presence'].copy()
    if vessel_features.shape != (
        NUM_VESSEL_NODES,
        len(EXPECTED_VESSEL_FEATURE_NAMES),
    ):
        raise RuntimeError(
            f'{subject_id}: unexpected vessel feature shape {vessel_features.shape}.'
        )
    if vessel_presence.shape != (NUM_VESSEL_NODES,):
        raise RuntimeError(
            f'{subject_id}: unexpected vessel presence shape {vessel_presence.shape}.'
        )
    if not cow_prior_valid:
        vessel_features.fill(0.0)
        vessel_presence.fill(0)
    vessel_feature_valid = np.broadcast_to(
        (vessel_presence > 0)[:, None], vessel_features.shape
    ).astype(np.uint8, copy=True)

    case_features = np.asarray(
        [
            float(atlas_enabled),
            registration_confidence,
            float(atlas_mode == 'syn'),
            float(atlas_mode == 'affine'),
            float(atlas_mode == 'disabled'),
            float(cow_prior_valid),
            float(arrays['lvo_present']),
            float(np.mean(vessel_presence > 0, dtype=np.float64)),
            float(np.mean(perfusion_valid[brain_mask], dtype=np.float64)),
            float(np.mean(pair_valid[brain_mask], dtype=np.float64)),
            float(np.log1p(brain_volume_ml)),
        ],
        dtype=np.float32,
    )

    finite_checks = [
        np.isfinite(node_features).all(),
        np.isfinite(node_reliability).all(),
        np.isfinite(case_features).all(),
        np.isfinite(vessel_features).all(),
    ]
    if not all(finite_checks):
        raise RuntimeError(f'{subject_id}: generated sidecar contains nonfinite values.')

    return {
        'node_features': node_features,
        'node_feature_valid': node_feature_valid,
        'node_valid': node_valid,
        'node_reliability': node_reliability,
        'node_voxels': node_voxels,
        'channel_sample_counts': channel_sample_counts,
        'case_features': case_features,
        'vessel_features': vessel_features.astype(np.float32, copy=False),
        'vessel_feature_valid': vessel_feature_valid,
        'vessel_presence': vessel_presence.astype(np.uint8, copy=False),
        'brain_volume_ml': brain_volume_ml,
        'atlas_valid_voxels': int(atlas_valid.sum()),
        'perfusion_valid_fraction': float(
            np.mean(perfusion_valid[brain_mask], dtype=np.float64)
        ),
        'pair_valid_fraction': float(
            np.mean(pair_valid[brain_mask], dtype=np.float64)
        ),
        'atlas_enabled': atlas_enabled,
        'atlas_mode': atlas_mode,
        'registration_confidence': registration_confidence,
        'cow_prior_valid': cow_prior_valid,
        'lvo_present': arrays['lvo_present'],
    }


def sidecar_arrays(subject_id, row, extracted):
    return {
        'subject_id': np.asarray(subject_id),
        'feature_version': np.asarray(FEATURE_VERSION),
        'code_revision': np.asarray(CODE_REVISION),
        'node_labels': node_labels,
        'node_features': extracted['node_features'],
        'node_feature_valid': extracted['node_feature_valid'],
        'node_valid': extracted['node_valid'],
        'node_reliability': extracted['node_reliability'],
        'node_voxels': extracted['node_voxels'],
        'channel_sample_counts': extracted['channel_sample_counts'],
        'node_feature_names': np.asarray(NODE_FEATURE_NAMES),
        'case_features': extracted['case_features'],
        'case_feature_names': np.asarray(CASE_FEATURE_NAMES),
        'vessel_features': extracted['vessel_features'],
        'vessel_feature_valid': extracted['vessel_feature_valid'],
        'vessel_presence': extracted['vessel_presence'],
        'vessel_names': np.asarray(EXPECTED_VESSEL_NAMES),
        'vessel_feature_names': np.asarray(EXPECTED_VESSEL_FEATURE_NAMES),
        'atlas_enabled': np.asarray(extracted['atlas_enabled'], dtype=np.uint8),
        'atlas_mode': np.asarray(extracted['atlas_mode']),
        'registration_confidence': np.asarray(
            extracted['registration_confidence'], dtype=np.float32
        ),
        'cow_prior_valid': np.asarray(
            extracted['cow_prior_valid'], dtype=np.uint8
        ),
        'lvo_present': np.asarray(extracted['lvo_present'], dtype=np.uint8),
        'brain_volume_ml': np.asarray(extracted['brain_volume_ml'], dtype=np.float32),
        'atlas_valid_voxels': np.asarray(
            extracted['atlas_valid_voxels'], dtype=np.int64
        ),
        'input_npz_sha256': np.asarray(str(row['trial8_npz_sha256'])),
        'selected_atlas_sha256': np.asarray(
            str(row['trial8_atlas_vascular30_sha256'])
            if extracted['atlas_enabled']
            else ''
        ),
        'graph_schema_sha256': np.asarray(graph_schema_sha256),
        'normalization_fitted': np.asarray(False, dtype=np.uint8),
        'target_arrays_read': np.asarray(False, dtype=np.uint8),
        'study_trainable_parameters': np.asarray(0, dtype=np.int64),
    }


def validate_sidecar(path, subject_id, expected_input_hash):
    with np.load(path, allow_pickle=False) as data:
        required = {
            'subject_id',
            'feature_version',
            'node_features',
            'node_feature_valid',
            'node_valid',
            'node_reliability',
            'node_voxels',
            'channel_sample_counts',
            'case_features',
            'vessel_features',
            'vessel_feature_valid',
            'vessel_presence',
            'input_npz_sha256',
            'graph_schema_sha256',
            'target_arrays_read',
        }
        if not required.issubset(data.files):
            return False, 'missing sidecar arrays'
        checks = [
            normalized_string(data['subject_id']) == subject_id,
            normalized_string(data['feature_version']) == FEATURE_VERSION,
            normalized_string(data['input_npz_sha256']) == expected_input_hash,
            normalized_string(data['graph_schema_sha256']) == graph_schema_sha256,
            data['node_features'].shape
            == (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES)),
            data['node_feature_valid'].shape
            == (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES)),
            data['node_valid'].shape == (NUM_TERRITORY_NODES,),
            data['node_reliability'].shape == (NUM_TERRITORY_NODES,),
            data['node_voxels'].shape == (NUM_TERRITORY_NODES,),
            data['channel_sample_counts'].shape
            == (NUM_TERRITORY_NODES, len(CONTINUOUS_CHANNELS)),
            data['case_features'].shape == (len(CASE_FEATURE_NAMES),),
            data['vessel_features'].shape
            == (NUM_VESSEL_NODES, len(EXPECTED_VESSEL_FEATURE_NAMES)),
            data['vessel_feature_valid'].shape
            == (NUM_VESSEL_NODES, len(EXPECTED_VESSEL_FEATURE_NAMES)),
            data['vessel_presence'].shape == (NUM_VESSEL_NODES,),
            np.isfinite(data['node_features']).all(),
            np.isfinite(data['node_reliability']).all(),
            np.isfinite(data['case_features']).all(),
            np.isfinite(data['vessel_features']).all(),
            not bool(np.asarray(data['target_arrays_read']).item()),
        ]
        if not all(checks):
            return False, 'sidecar validation check failed'
    return True, ''






audit_rows = []
error_rows = []
sorted_manifest = manifest_df.sort_values('subject_id').reset_index(drop=True)

for case_index, row in sorted_manifest.iterrows():
    subject_id = str(row['subject_id'])
    sidecar_path = SIDECAR_DIR / f'{subject_id}_{FEATURE_VERSION}.npz'
    completion_path = COMPLETION_DIR / f'{subject_id}_completion.json'
    status = 'generated'

    try:
        resume_valid = False
        if sidecar_path.exists() and completion_path.exists():
            completion = json.loads(completion_path.read_text(encoding='utf-8'))
            completion_checks = [
                completion.get('subject_id') == subject_id,
                completion.get('feature_version') == FEATURE_VERSION,
                completion.get('input_npz_sha256') == str(row['trial8_npz_sha256']),
                completion.get('graph_schema_sha256') == graph_schema_sha256,
                completion.get('sidecar_sha256') == file_sha256(sidecar_path),
                completion.get('target_arrays_read') is False,
            ]
            sidecar_valid, _ = validate_sidecar(
                sidecar_path, subject_id, str(row['trial8_npz_sha256'])
            )
            resume_valid = bool(all(completion_checks) and sidecar_valid)

        if resume_valid:
            status = 'resumed'
            with np.load(sidecar_path, allow_pickle=False) as data:
                atlas_enabled = bool(np.asarray(data['atlas_enabled']).item())
                atlas_mode = normalized_string(data['atlas_mode'])
                cow_prior_valid = bool(np.asarray(data['cow_prior_valid']).item())
                lvo_present = bool(np.asarray(data['lvo_present']).item())
                registration_confidence = float(
                    np.asarray(data['registration_confidence']).item()
                )
                valid_nodes = int(np.asarray(data['node_valid']).sum())
                reliable_nodes = int(
                    (np.asarray(data['node_reliability']) >= 0.5).sum()
                )
                present_vessels = int(np.asarray(data['vessel_presence']).sum())
                brain_volume_ml = float(np.asarray(data['brain_volume_ml']).item())
                atlas_valid_voxels = int(
                    np.asarray(data['atlas_valid_voxels']).item()
                )
                case_values = np.asarray(data['case_features'], dtype=np.float32)
                perfusion_valid_fraction = float(
                    case_values[CASE_FEATURE_NAMES.index('perfusion_valid_fraction')]
                )
                pair_valid_fraction = float(
                    case_values[
                        CASE_FEATURE_NAMES.index(
                            'contralateral_pair_valid_fraction'
                        )
                    ]
                )
        else:
            arrays = load_subject_inputs(row)
            extracted = extract_subject_features(row, arrays)
            atomic_write_npz(
                sidecar_path,
                **sidecar_arrays(subject_id, row, extracted),
            )
            sidecar_valid, validation_error = validate_sidecar(
                sidecar_path, subject_id, str(row['trial8_npz_sha256'])
            )
            if not sidecar_valid:
                raise RuntimeError(validation_error)

            sidecar_hash = file_sha256(sidecar_path)
            completion_record = {
                'created_at_utc': datetime.now(timezone.utc).isoformat(),
                'subject_id': subject_id,
                'dev_split': str(row['dev_split']),
                'feature_version': FEATURE_VERSION,
                'code_revision': CODE_REVISION,
                'input_npz_path': str(row['vascufate_v2_npz_path']),
                'input_npz_sha256': str(row['trial8_npz_sha256']),
                'graph_schema_sha256': graph_schema_sha256,
                'sidecar_path': str(sidecar_path),
                'sidecar_sha256': sidecar_hash,
                'target_arrays_read': False,
                'study_trainable_parameters': 0,
            }
            atomic_write_json(completion_record, completion_path)

            atlas_enabled = extracted['atlas_enabled']
            atlas_mode = extracted['atlas_mode']
            cow_prior_valid = extracted['cow_prior_valid']
            lvo_present = extracted['lvo_present']
            registration_confidence = extracted['registration_confidence']
            valid_nodes = int(extracted['node_valid'].sum())
            reliable_nodes = int((extracted['node_reliability'] >= 0.5).sum())
            present_vessels = int(extracted['vessel_presence'].sum())
            brain_volume_ml = extracted['brain_volume_ml']
            atlas_valid_voxels = extracted['atlas_valid_voxels']
            perfusion_valid_fraction = extracted['perfusion_valid_fraction']
            pair_valid_fraction = extracted['pair_valid_fraction']

        sidecar_hash = file_sha256(sidecar_path)
        audit_rows.append({
            'subject_id': subject_id,
            'dev_split': str(row['dev_split']),
            'status': status,
            'atlas_mode': atlas_mode,
            'atlas_enabled': atlas_enabled,
            'registration_confidence': registration_confidence,
            'cow_prior_valid': cow_prior_valid,
            'lvo_present': lvo_present,
            'valid_territory_nodes': valid_nodes,
            'reliable_territory_nodes_ge_0p5': reliable_nodes,
            'present_vessels': present_vessels,
            'brain_volume_ml': brain_volume_ml,
            'atlas_valid_voxels': atlas_valid_voxels,
            'perfusion_valid_fraction': perfusion_valid_fraction,
            'contralateral_pair_valid_fraction': pair_valid_fraction,
            'node_feature_shape': str(
                (NUM_TERRITORY_NODES, len(NODE_FEATURE_NAMES))
            ),
            'vessel_feature_shape': str(
                (NUM_VESSEL_NODES, len(EXPECTED_VESSEL_FEATURE_NAMES))
            ),
            'sidecar_path': str(sidecar_path),
            'sidecar_sha256': sidecar_hash,
            'finite': True,
            'target_arrays_read': False,
            'study_trainable_parameters': 0,
        })

        completed = case_index + 1
        if completed == 1 or completed % 10 == 0 or completed == EXPECTED_SUBJECTS:
            print(
                f'Completed {completed}/{EXPECTED_SUBJECTS}: {subject_id}, '
                f'mode={atlas_mode}, nodes={valid_nodes}, status={status}'
            )

    except Exception as error:
        error_rows.append({
            'subject_id': subject_id,
            'dev_split': str(row['dev_split']),
            'error': f'{type(error).__name__}: {error}',
        })
        print(
            f'Failed {case_index + 1}/{EXPECTED_SUBJECTS}: '
            f'{subject_id}: {type(error).__name__}: {error}'
        )

audit_df = pd.DataFrame(audit_rows)
if len(audit_df):
    audit_df = audit_df.sort_values('subject_id').reset_index(drop=True)
atomic_write_csv(audit_df, AUDIT_CSV_PATH)

error_df = pd.DataFrame(error_rows, columns=['subject_id', 'dev_split', 'error'])
atomic_write_csv(error_df, ERROR_CSV_PATH)

all_subjects_generated = (
    len(audit_df) == EXPECTED_SUBJECTS
    and audit_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    and len(error_df) == 0
)
mode_counts_match = (
    all_subjects_generated
    and audit_df['atlas_mode'].value_counts().to_dict() == EXPECTED_MODES
)
disabled_nodes_zero = (
    all_subjects_generated
    and (
        audit_df.loc[
            audit_df['atlas_mode'] == 'disabled', 'valid_territory_nodes'
        ]
        == 0
    ).all()
)
enabled_nodes_present = (
    all_subjects_generated
    and (
        audit_df.loc[
            audit_df['atlas_mode'] != 'disabled', 'valid_territory_nodes'
        ]
        > 0
    ).all()
)
all_finite = all_subjects_generated and audit_df['finite'].map(as_bool).all()
target_free = all_subjects_generated and not audit_df[
    'target_arrays_read'
].map(as_bool).any()

generation_checks = {
    'frozen_manifest_hash_matches': observed_manifest_sha256
    == EXPECTED_MANIFEST_SHA256,
    'graph_schema_written': GRAPH_SCHEMA_NPZ_PATH.exists(),
    'graph_schema_json_written': GRAPH_SCHEMA_JSON_PATH.exists(),
    'all_148_subjects_generated': all_subjects_generated,
    'mode_counts_match_frozen_policy': mode_counts_match,
    'disabled_subject_territory_nodes_are_zero': disabled_nodes_zero,
    'enabled_subjects_have_territory_nodes': enabled_nodes_present,
    'all_sidecars_are_finite': all_finite,
    'target_arrays_not_read': target_free,
    'study_trainable_parameters_are_zero': True,
}
generation_passed = bool(all(generation_checks.values()))

if generation_passed:
    sidecar_lookup = audit_df.set_index('subject_id')
    candidate_manifest_df = manifest_df.copy()
    candidate_manifest_df['trial8_graph_feature_version'] = FEATURE_VERSION
    candidate_manifest_df['trial8_graph_sidecar_path'] = (
        candidate_manifest_df['subject_id'].map(sidecar_lookup['sidecar_path'])
    )
    candidate_manifest_df['trial8_graph_sidecar_sha256'] = (
        candidate_manifest_df['subject_id'].map(sidecar_lookup['sidecar_sha256'])
    )
    candidate_manifest_df['trial8_graph_schema_path'] = str(GRAPH_SCHEMA_NPZ_PATH)
    candidate_manifest_df['trial8_graph_schema_sha256'] = graph_schema_sha256
    candidate_manifest_df['trial8_graph_feature_generation_complete'] = True
    candidate_manifest_df['trial8_graph_feature_verification_passed'] = False
    candidate_manifest_df['trial8_model_training_authorized'] = False
    atomic_write_csv(candidate_manifest_df, CANDIDATE_MANIFEST_PATH)
    candidate_manifest_sha256 = file_sha256(CANDIDATE_MANIFEST_PATH)
else:
    candidate_manifest_sha256 = ''

summary_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'feature_version': FEATURE_VERSION,
    'code_revision': CODE_REVISION,
    'generation_passed': generation_passed,
    'checks': generation_checks,
    'subjects_expected': EXPECTED_SUBJECTS,
    'subjects_generated': int(len(audit_df)),
    'subjects_failed': int(len(error_df)),
    'mode_counts': audit_df['atlas_mode'].value_counts().to_dict()
    if len(audit_df)
    else {},
    'territory_node_count': NUM_TERRITORY_NODES,
    'territory_feature_count': len(NODE_FEATURE_NAMES),
    'vessel_node_count': NUM_VESSEL_NODES,
    'vessel_feature_count': len(EXPECTED_VESSEL_FEATURE_NAMES),
    'graph_schema_path': str(GRAPH_SCHEMA_NPZ_PATH),
    'graph_schema_sha256': graph_schema_sha256,
    'audit_csv_path': str(AUDIT_CSV_PATH),
    'error_csv_path': str(ERROR_CSV_PATH),
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH)
    if generation_passed
    else '',
    'candidate_manifest_sha256': candidate_manifest_sha256,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'model_training_authorized': False,
}
atomic_write_json(summary_record, SUMMARY_JSON_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'feature_version': FEATURE_VERSION,
    'generation_passed': generation_passed,
    'checks': generation_checks,
    'frozen_manifest_path': str(FROZEN_MANIFEST_PATH),
    'frozen_manifest_sha256': observed_manifest_sha256,
    'graph_schema_path': str(GRAPH_SCHEMA_NPZ_PATH),
    'graph_schema_sha256': graph_schema_sha256,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH)
    if generation_passed
    else '',
    'candidate_manifest_sha256': candidate_manifest_sha256,
    'graph_feature_verification_passed': False,
    'model_training_authorized': False,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(gate_record, GENERATION_GATE_PATH)

print('\nGeneration checks:')
display(
    pd.DataFrame(
        [
            {'check': name, 'passed': passed}
            for name, passed in generation_checks.items()
        ]
    )
)

if len(audit_df):
    print('\nSidecar summary by atlas mode:')
    display(
        audit_df.groupby('atlas_mode', as_index=False).agg(
            subjects=('subject_id', 'count'),
            mean_valid_territory_nodes=('valid_territory_nodes', 'mean'),
            minimum_valid_territory_nodes=('valid_territory_nodes', 'min'),
            mean_present_vessels=('present_vessels', 'mean'),
            mean_registration_confidence=('registration_confidence', 'mean'),
        )
    )

if not generation_passed:
    if len(error_df):
        print('\nGeneration errors:')
        display(error_df)
    failed_checks = [name for name, passed in generation_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 GRAPH SIDECAR GENERATION FAILED. Do not continue to graph '
        'verification or model training. Failed checks: '
        + ', '.join(failed_checks)
    )

print('\n' + '=' * 80)
print('TRIAL8 TARGET-FREE VASCULAR GRAPH SIDECAR GENERATION PASSED')
print('=' * 80)
print(f'Subjects generated:                  {len(audit_df)}')
print(f'Territory nodes per subject:        {NUM_TERRITORY_NODES}')
print(f'Territory features per node:        {len(NODE_FEATURE_NAMES)}')
print(f'Vessel nodes per subject:           {NUM_VESSEL_NODES}')
print(f'Vessel features per node:           {len(EXPECTED_VESSEL_FEATURE_NAMES)}')
print(f'Territory graph edges:              {territory_edge_index.shape[1]}')
print(f'Vessel graph edges:                 {vessel_edge_index.shape[1]}')
print(f'Vessel-to-territory edges:          {cross_edge_index.shape[1]}')
print('Target arrays read:                 False')
print('Study trainable parameters:         0 (feature generation only)')
print('Graph verification passed:          False')
print('Model training authorized:          False')
print(f'Graph schema:                       {GRAPH_SCHEMA_NPZ_PATH}')
print(f'Generation audit:                   {AUDIT_CSV_PATH}')
print(f'Candidate manifest:                 {CANDIDATE_MANIFEST_PATH}')
print(f'Generation gate:                    {GENERATION_GATE_PATH}')
print('\nNext: run the standalone graph-sidecar verification cell.')


In [ ]:
import sys
import os
import json
import hashlib
import subprocess
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
CODE_REVISION = 'r1_independent_graph_sidecar_verification'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_MODES = {'syn': 139, 'affine': 2, 'disabled': 7}
EXPECTED_SOURCE_MANIFEST_SHA256 = (
    '61b58b16cd28313aa2fbfea6591facb4d504b56241fa9b8789c3c5a755244cd5'
)

NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
MIN_STATS_VOXELS = 16
NODE_RELIABILITY_FULL_SUPPORT_VOXELS = 256
FEATURE_ATOL = 2e-5
FEATURE_RTOL = 2e-5

GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
GENERATION_GATE_PATH = GRAPH_ROOT / 'graph_sidecar_generation_gate.json'
GENERATION_SUMMARY_PATH = GRAPH_ROOT / 'graph_sidecar_generation_summary.json'
GRAPH_SCHEMA_PATH = GRAPH_ROOT / 'graph_schema.npz'
GRAPH_SCHEMA_JSON_PATH = GRAPH_ROOT / 'graph_schema.json'
CANDIDATE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_candidate.csv'
)

VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFIED_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)
CASE_AUDIT_PATH = VERIFICATION_ROOT / 'graph_sidecar_case_audit.csv'
ERROR_AUDIT_PATH = VERIFICATION_ROOT / 'graph_sidecar_errors.csv'
TOPOLOGY_AUDIT_PATH = VERIFICATION_ROOT / 'graph_topology_audit.csv'
TERRITORY_DISTRIBUTION_PATH = (
    VERIFICATION_ROOT / 'territory_feature_distribution_by_split.csv'
)
VESSEL_DISTRIBUTION_PATH = (
    VERIFICATION_ROOT / 'vessel_feature_distribution_by_split.csv'
)
CASE_DISTRIBUTION_PATH = (
    VERIFICATION_ROOT / 'case_feature_distribution_by_split.csv'
)
VERIFICATION_SUMMARY_PATH = VERIFICATION_ROOT / 'verification_summary.json'
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
CONTINUOUS_CHANNELS = EXPECTED_INPUT_CHANNELS[:13]
STAT_NAMES = ['mean', 'std', 'p10', 'p90']

EXPECTED_VESSEL_NAMES = [
    'BA',
    'R_PCA',
    'L_PCA',
    'R_ICA',
    'R_MCA',
    'L_ICA',
    'L_MCA',
    'R_Pcom',
    'L_Pcom',
    'Acom',
    'R_ACA',
    'L_ACA',
    'third_A2',
]

EXPECTED_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

META_FEATURE_NAMES = [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in CONTINUOUS_CHANNELS
    for stat in STAT_NAMES
] + META_FEATURE_NAMES

EXPECTED_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]

EXPECTED_TERRITORY_RELATIONS = [
    'self',
    'template_contact_26n',
    'contralateral_homolog',
    'same_unilateral_parent',
]
EXPECTED_VESSEL_RELATIONS = [
    'self',
    'anatomical_connection',
    'contralateral_homolog',
]
EXPECTED_CROSS_RELATIONS = [
    'primary_supply',
    'communicating_or_proximal_supply',
]

FORBIDDEN_SIDECAR_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}

SOURCE_KEYS_FOR_ALL_CASES = {
    'subject_id',
    'lvo_present',
    'vessel_features_raw',
    'vessel_presence',
    'vessel_names',
    'vessel_feature_names',
}

SOURCE_KEYS_FOR_RECOMPUTE = {
    'image',
    'perfusion_valid',
    'mtt_valid',
    'mtt_interpolation_support',
    'cow_label',
    'cow_soft_union',
    'lvo_proximity',
    'affine',
    'input_channels',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


for path in [VERIFICATION_ROOT, VERIFIED_MANIFEST_PATH.parent]:
    assert_project_path_is_safe(path)
    path.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary_path = Path(str(path) + '.tmp')
    temporary_path.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary_path, path)


def atomic_write_csv(frame, path):
    temporary_path = Path(str(path) + '.tmp')
    frame.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def atomic_write_npz(path, **arrays):
    temporary_path = Path(str(path) + '.tmp')
    with temporary_path.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary_path, path)


def decode_strings(array):
    return [str(value) for value in np.asarray(array).tolist()]


def scalar_string(array):
    return str(np.asarray(array).item())


def load_effective_3d(path):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    elif array.ndim != 3:
        raise RuntimeError(f'Unsupported image shape: {path} -> {array.shape}')
    return image, array


def binary_array(array):
    values = set(int(value) for value in np.unique(array))
    return values.issubset({0, 1})


def safe_stats(values):
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]
    if values.size < MIN_STATS_VOXELS:
        return np.zeros(4, dtype=np.float32), False, int(values.size)
    percentiles = np.percentile(values, [10.0, 90.0]).astype(np.float32)
    result = np.asarray(
        [
            np.mean(values, dtype=np.float64),
            np.std(values, dtype=np.float64),
            percentiles[0],
            percentiles[1],
        ],
        dtype=np.float32,
    )
    return result, bool(np.isfinite(result).all()), int(values.size)






required_paths = [
    GENERATION_GATE_PATH,
    GENERATION_SUMMARY_PATH,
    GRAPH_SCHEMA_PATH,
    GRAPH_SCHEMA_JSON_PATH,
    CANDIDATE_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 graph-verification prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

generation_gate = json.loads(GENERATION_GATE_PATH.read_text(encoding='utf-8'))
generation_summary = json.loads(
    GENERATION_SUMMARY_PATH.read_text(encoding='utf-8')
)

candidate_manifest_sha256 = file_sha256(CANDIDATE_MANIFEST_PATH)
graph_schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)

generation_checks = {
    'generation_gate_passed': generation_gate.get('generation_passed') is True,
    'generation_summary_passed': generation_summary.get('generation_passed')
    is True,
    'source_manifest_hash_matches': generation_gate.get('frozen_manifest_sha256')
    == EXPECTED_SOURCE_MANIFEST_SHA256,
    'candidate_manifest_hash_matches': generation_gate.get(
        'candidate_manifest_sha256'
    )
    == candidate_manifest_sha256,
    'graph_schema_hash_matches': generation_gate.get('graph_schema_sha256')
    == graph_schema_sha256,
    'verification_was_not_preapproved': generation_gate.get(
        'graph_feature_verification_passed'
    )
    is False,
    'training_was_locked': generation_gate.get('model_training_authorized')
    is False,
    'generation_target_free': generation_gate.get('target_arrays_read') is False,
    'generation_trainable_parameters_zero': generation_gate.get(
        'study_trainable_parameters'
    )
    == 0,
}
if not all(generation_checks.values()):
    failed = [name for name, passed in generation_checks.items() if not passed]
    raise RuntimeError('Graph-generation gate checks failed: ' + ', '.join(failed))

manifest_df = pd.read_csv(CANDIDATE_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'cow_prior_valid',
    'trial8_npz_sha256',
    'trial8_synthstrip_mask_path',
    'trial8_synthstrip_mask_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_registration_confidence',
    'trial8_atlas_vascular30_path',
    'trial8_atlas_vascular30_sha256',
    'trial8_graph_feature_version',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_generation_complete',
    'trial8_graph_feature_verification_passed',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Candidate manifest is missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_are_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'mode_counts_match': manifest_df['trial8_atlas_mode'].value_counts().to_dict()
    == EXPECTED_MODES,
    'feature_version_matches': (
        manifest_df['trial8_graph_feature_version'] == FEATURE_VERSION
    ).all(),
    'generation_complete_for_all': manifest_df[
        'trial8_graph_feature_generation_complete'
    ].map(as_bool).all(),
    'verification_false_before_this_cell': ~manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).any(),
    'training_locked_before_this_cell': ~manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).any(),
    'schema_hash_matches_for_all': (
        manifest_df['trial8_graph_schema_sha256'] == graph_schema_sha256
    ).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Candidate manifest checks failed: ' + ', '.join(failed))







with np.load(GRAPH_SCHEMA_PATH, allow_pickle=True) as source_schema:
    source_schema_arrays = {
        name: np.asarray(source_schema[name]) for name in source_schema.files
    }

object_schema_array_names = []
safe_schema_arrays = {}
for name, array in source_schema_arrays.items():
    if array.dtype.kind == 'O':
        object_schema_array_names.append(name)
        flattened = array.reshape(-1).tolist()
        if not all(isinstance(value, (str, np.str_)) for value in flattened):
            raise RuntimeError(
                f'Graph schema object array {name!r} contains a non-string '
                'value and cannot be safely converted.'
            )
        safe_schema_arrays[name] = array.astype(np.str_)
    elif array.dtype.kind in {'b', 'i', 'u', 'f', 'S', 'U'}:
        safe_schema_arrays[name] = array
    else:
        raise RuntimeError(
            f'Graph schema array {name!r} has unsupported dtype {array.dtype}.'
        )

atomic_write_npz(VERIFIED_GRAPH_SCHEMA_PATH, **safe_schema_arrays)
verified_graph_schema_sha256 = file_sha256(VERIFIED_GRAPH_SCHEMA_PATH)

with np.load(VERIFIED_GRAPH_SCHEMA_PATH, allow_pickle=False) as safe_schema:
    schema_arrays = {
        name: np.asarray(safe_schema[name]) for name in safe_schema.files
    }

schema_conversion_checks = {
    'source_schema_hash_was_verified_before_conversion': generation_checks[
        'graph_schema_hash_matches'
    ],
    'object_arrays_were_string_only': all(
        source_schema_arrays[name].dtype.kind != 'O'
        or all(
            isinstance(value, (str, np.str_))
            for value in source_schema_arrays[name].reshape(-1).tolist()
        )
        for name in source_schema_arrays
    ),
    'safe_schema_contains_no_object_arrays': not any(
        array.dtype.kind == 'O' for array in schema_arrays.values()
    ),
    'safe_schema_array_names_preserved': set(schema_arrays)
    == set(source_schema_arrays),
    'safe_schema_values_preserved': all(
        np.array_equal(
            schema_arrays[name],
            source_schema_arrays[name].astype(np.str_)
            if source_schema_arrays[name].dtype.kind == 'O'
            else source_schema_arrays[name],
        )
        for name in source_schema_arrays
    ),
}
if not all(schema_conversion_checks.values()):
    failed = [
        name for name, passed in schema_conversion_checks.items() if not passed
    ]
    raise RuntimeError('Safe graph-schema conversion failed: ' + ', '.join(failed))

required_schema_arrays = {
    'feature_version',
    'node_labels',
    'node_names',
    'node_acronyms',
    'node_hemisphere',
    'node_parent_labels',
    'node_feature_names',
    'case_feature_names',
    'vessel_names',
    'vessel_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(required_schema_arrays - set(schema_arrays))
if missing_schema_arrays:
    raise RuntimeError(f'Graph schema is missing arrays: {missing_schema_arrays}')


def edge_checks(prefix, edge_index, edge_type, edge_weight, nodes, relations):
    shape_ok = (
        edge_index.ndim == 2
        and edge_index.shape[0] == 2
        and edge_type.shape == (edge_index.shape[1],)
        and edge_weight.shape == (edge_index.shape[1],)
    )
    if not shape_ok:
        return {
            f'{prefix}_edge_shapes_valid': False,
            f'{prefix}_edge_indices_in_range': False,
            f'{prefix}_edge_types_in_range': False,
            f'{prefix}_edge_weights_valid': False,
            f'{prefix}_edges_unique': False,
        }
    triples = [
        (int(edge_index[0, i]), int(edge_index[1, i]), int(edge_type[i]))
        for i in range(edge_index.shape[1])
    ]
    return {
        f'{prefix}_edge_shapes_valid': True,
        f'{prefix}_edge_indices_in_range': bool(
            edge_index.min() >= 0 and edge_index.max() < nodes
        ),
        f'{prefix}_edge_types_in_range': bool(
            edge_type.min() >= 0 and edge_type.max() < relations
        ),
        f'{prefix}_edge_weights_valid': bool(
            np.isfinite(edge_weight).all()
            and (edge_weight > 0).all()
            and (edge_weight <= 1.0 + 1e-6).all()
        ),
        f'{prefix}_edges_unique': len(triples) == len(set(triples)),
    }


territory_edge_index = schema_arrays['territory_edge_index'].astype(np.int64)
territory_edge_type = schema_arrays['territory_edge_type'].astype(np.int64)
territory_edge_weight = schema_arrays['territory_edge_weight'].astype(np.float64)
vessel_edge_index = schema_arrays['vessel_edge_index'].astype(np.int64)
vessel_edge_type = schema_arrays['vessel_edge_type'].astype(np.int64)
vessel_edge_weight = schema_arrays['vessel_edge_weight'].astype(np.float64)
cross_edge_index = schema_arrays['cross_edge_index'].astype(np.int64)
cross_edge_type = schema_arrays['cross_edge_type'].astype(np.int64)
cross_edge_weight = schema_arrays['cross_edge_weight'].astype(np.float64)

topology_checks = {
    'feature_version_matches': scalar_string(schema_arrays['feature_version'])
    == FEATURE_VERSION,
    'node_labels_are_1_to_30': np.array_equal(
        schema_arrays['node_labels'], np.arange(1, 31, dtype=np.int16)
    ),
    'node_names_are_unique': len(set(decode_strings(schema_arrays['node_names'])))
    == NUM_TERRITORY_NODES,
    'node_acronyms_are_unique': len(
        set(decode_strings(schema_arrays['node_acronyms']))
    )
    == NUM_TERRITORY_NODES,
    'hemispheres_are_left_right': set(
        decode_strings(schema_arrays['node_hemisphere'])
    )
    == {'left', 'right'},
    'node_features_match_frozen_schema': decode_strings(
        schema_arrays['node_feature_names']
    )
    == EXPECTED_NODE_FEATURE_NAMES,
    'case_features_match_frozen_schema': decode_strings(
        schema_arrays['case_feature_names']
    )
    == EXPECTED_CASE_FEATURE_NAMES,
    'vessel_names_match_frozen_schema': decode_strings(
        schema_arrays['vessel_names']
    )
    == EXPECTED_VESSEL_NAMES,
    'vessel_features_match_frozen_schema': decode_strings(
        schema_arrays['vessel_feature_names']
    )
    == EXPECTED_VESSEL_FEATURE_NAMES,
    'territory_relations_match': decode_strings(
        schema_arrays['territory_relation_names']
    )
    == EXPECTED_TERRITORY_RELATIONS,
    'vessel_relations_match': decode_strings(
        schema_arrays['vessel_relation_names']
    )
    == EXPECTED_VESSEL_RELATIONS,
    'cross_relations_match': decode_strings(schema_arrays['cross_relation_names'])
    == EXPECTED_CROSS_RELATIONS,
    **edge_checks(
        'territory',
        territory_edge_index,
        territory_edge_type,
        territory_edge_weight,
        NUM_TERRITORY_NODES,
        len(EXPECTED_TERRITORY_RELATIONS),
    ),
    **edge_checks(
        'vessel',
        vessel_edge_index,
        vessel_edge_type,
        vessel_edge_weight,
        NUM_VESSEL_NODES,
        len(EXPECTED_VESSEL_RELATIONS),
    ),
    **edge_checks(
        'cross',
        cross_edge_index,
        cross_edge_type,
        cross_edge_weight,
        max(NUM_VESSEL_NODES, NUM_TERRITORY_NODES),
        len(EXPECTED_CROSS_RELATIONS),
    ),
}


topology_checks['cross_source_indices_in_vessel_range'] = bool(
    cross_edge_index[0].min() >= 0
    and cross_edge_index[0].max() < NUM_VESSEL_NODES
)
topology_checks['cross_target_indices_in_territory_range'] = bool(
    cross_edge_index[1].min() >= 0
    and cross_edge_index[1].max() < NUM_TERRITORY_NODES
)


def directed_relation_set(edge_index, edge_type, relation):
    mask = edge_type == relation
    return set(
        (int(source), int(target))
        for source, target in edge_index[:, mask].T
    )


territory_self = directed_relation_set(territory_edge_index, territory_edge_type, 0)
vessel_self = directed_relation_set(vessel_edge_index, vessel_edge_type, 0)
territory_homolog = directed_relation_set(
    territory_edge_index, territory_edge_type, 2
)
territory_parent = directed_relation_set(
    territory_edge_index, territory_edge_type, 3
)

expected_territory_self = {(index, index) for index in range(30)}
expected_vessel_self = {(index, index) for index in range(13)}
expected_homolog = set()
for left_index in range(0, 30, 2):
    expected_homolog.add((left_index, left_index + 1))
    expected_homolog.add((left_index + 1, left_index))

expected_parent = set()
parent_labels = schema_arrays['node_parent_labels'].astype(int)
for parent in sorted(set(parent_labels.tolist())):
    indices = np.flatnonzero(parent_labels == parent).tolist()
    for first, second in combinations(indices, 2):
        expected_parent.add((first, second))
        expected_parent.add((second, first))

topology_checks.update({
    'territory_self_loops_exact': territory_self == expected_territory_self,
    'vessel_self_loops_exact': vessel_self == expected_vessel_self,
    'all_15_homolog_pairs_present': territory_homolog == expected_homolog,
    'same_parent_edges_exact': territory_parent == expected_parent,
})


def directed_graph_is_symmetric(edge_index, edge_type, edge_weight):
    lookup = {}
    for index in range(edge_index.shape[1]):
        key = (
            int(edge_index[0, index]),
            int(edge_index[1, index]),
            int(edge_type[index]),
        )
        lookup[key] = float(edge_weight[index])
    for (source, target, relation), weight in lookup.items():
        reverse = (target, source, relation)
        if reverse not in lookup or not np.isclose(
            weight, lookup[reverse], atol=1e-7, rtol=1e-7
        ):
            return False
    return True


topology_checks['territory_graph_is_directed_symmetric'] = (
    directed_graph_is_symmetric(
        territory_edge_index, territory_edge_type, territory_edge_weight
    )
)
topology_checks['vessel_graph_is_directed_symmetric'] = directed_graph_is_symmetric(
    vessel_edge_index, vessel_edge_type, vessel_edge_weight
)

topology_df = pd.DataFrame(
    [{'check': name, 'passed': passed} for name, passed in topology_checks.items()]
)
atomic_write_csv(topology_df, TOPOLOGY_AUDIT_PATH)
if not all(topology_checks.values()):
    failed = [name for name, passed in topology_checks.items() if not passed]
    raise RuntimeError('Graph-topology checks failed: ' + ', '.join(failed))

print('Trial8 graph-generation and topology gates passed.')
print(f'Candidate subjects: {len(manifest_df)}')
print(f'Source graph schema SHA-256: {graph_schema_sha256}')
print(f'Pickle-free graph schema SHA-256: {verified_graph_schema_sha256}')
print(f'Safely converted object arrays: {object_schema_array_names}')
print('Target arrays read: False')
print('Study trainable parameters: 0 (verification only)')






recompute_subjects = set(
    manifest_df.loc[
        manifest_df['trial8_atlas_mode'].isin(['affine', 'disabled']),
        'subject_id',
    ].astype(str)
)
for split in ['train', 'val', 'test']:
    split_syn = sorted(
        manifest_df.loc[
            (manifest_df['dev_split'] == split)
            & (manifest_df['trial8_atlas_mode'] == 'syn'),
            'subject_id',
        ].astype(str)
    )
    for index in sorted({0, len(split_syn) // 2, len(split_syn) - 1}):
        recompute_subjects.add(split_syn[index])

print(f'Deterministic semantic recomputation subjects: {len(recompute_subjects)}')
print(sorted(recompute_subjects))






def recompute_subject(row, sidecar):
    subject_id = str(row['subject_id'])
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    cow_prior_valid = as_bool(row['cow_prior_valid'])

    if not atlas_enabled:
        return {
            'node_features': np.zeros(
                (NUM_TERRITORY_NODES, len(EXPECTED_NODE_FEATURE_NAMES)),
                dtype=np.float32,
            ),
            'node_feature_valid': np.zeros(
                (NUM_TERRITORY_NODES, len(EXPECTED_NODE_FEATURE_NAMES)),
                dtype=np.uint8,
            ),
            'node_valid': np.zeros(NUM_TERRITORY_NODES, dtype=np.uint8),
            'node_voxels': np.zeros(NUM_TERRITORY_NODES, dtype=np.int32),
            'channel_sample_counts': np.zeros(
                (NUM_TERRITORY_NODES, len(CONTINUOUS_CHANNELS)),
                dtype=np.int32,
            ),
            'case_features': None,
        }

    npz_path = Path(str(row['vascufate_v2_npz_path']))
    brain_mask_path = Path(str(row['trial8_synthstrip_mask_path']))
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    for path in [npz_path, brain_mask_path, atlas_path]:
        if not path.exists():
            raise FileNotFoundError(f'{subject_id}: missing recomputation input {path}')

    with np.load(npz_path, allow_pickle=False) as source:
        required = SOURCE_KEYS_FOR_ALL_CASES | SOURCE_KEYS_FOR_RECOMPUTE
        missing = sorted(required - set(source.files))
        if missing:
            raise RuntimeError(f'{subject_id}: source NPZ missing {missing}')
        image = np.asarray(source['image'], dtype=np.float32)
        perfusion_valid = np.asarray(source['perfusion_valid']) > 0
        mtt_valid = np.asarray(source['mtt_valid']) > 0
        mtt_support = np.asarray(
            source['mtt_interpolation_support'], dtype=np.float32
        )
        cow_label = np.asarray(source['cow_label'])
        cow_soft_union = np.asarray(source['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(source['lvo_proximity'], dtype=np.float32)
        affine = np.asarray(source['affine'], dtype=np.float64)
        input_channels = decode_strings(source['input_channels'])

    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: source input-channel schema changed.')

    brain_image, brain_array = load_effective_3d(brain_mask_path)
    atlas_image, atlas_array = load_effective_3d(atlas_path)
    brain_mask = brain_array > 0.5
    atlas_labels = np.rint(atlas_array).astype(np.int16)
    if image.shape[1:] != brain_mask.shape or brain_mask.shape != atlas_labels.shape:
        raise RuntimeError(f'{subject_id}: recomputation geometry mismatch.')
    for role, image_object in [('brain', brain_image), ('atlas', atlas_image)]:
        if not np.allclose(
            image_object.affine, affine, atol=1e-4, rtol=1e-6
        ):
            raise RuntimeError(f'{subject_id}: {role} affine mismatch.')

    pair_valid = image[14] > 0.5
    voxel_sizes = np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0))
    voxel_volume_ml = float(np.prod(voxel_sizes) / 1000.0)
    brain_voxels = int(brain_mask.sum())
    brain_volume_ml = brain_voxels * voxel_volume_ml

    node_features = np.zeros(
        (NUM_TERRITORY_NODES, len(EXPECTED_NODE_FEATURE_NAMES)), dtype=np.float32
    )
    node_feature_valid = np.zeros_like(node_features, dtype=np.uint8)
    node_valid = np.zeros(NUM_TERRITORY_NODES, dtype=np.uint8)
    node_voxels = np.zeros(NUM_TERRITORY_NODES, dtype=np.int32)
    sample_counts = np.zeros(
        (NUM_TERRITORY_NODES, len(CONTINUOUS_CHANNELS)), dtype=np.int32
    )

    for node_index, territory_label in enumerate(range(1, 31)):
        territory_mask = atlas_labels == territory_label
        count = int(territory_mask.sum())
        node_voxels[node_index] = count
        if count == 0:
            continue
        node_valid[node_index] = 1
        offset = 0
        for channel_index, channel_name in enumerate(CONTINUOUS_CHANNELS):
            measurement_mask = territory_mask
            if channel_name in {'cbf', 'cbv', 'mtt', 'tmax'}:
                measurement_mask = measurement_mask & perfusion_valid
            elif channel_name.startswith('relative_') or channel_name.startswith(
                'delta_'
            ):
                measurement_mask = measurement_mask & pair_valid
            if channel_name == 'mtt':
                measurement_mask = measurement_mask & mtt_valid
            stats, valid, sample_count = safe_stats(
                image[channel_index][measurement_mask]
            )
            node_features[node_index, offset : offset + 4] = stats
            node_feature_valid[node_index, offset : offset + 4] = int(valid)
            sample_counts[node_index, channel_index] = sample_count
            offset += 4

        meta = np.asarray(
            [
                np.log1p(count),
                np.log1p(count * voxel_volume_ml),
                count / brain_voxels,
                np.mean(perfusion_valid[territory_mask], dtype=np.float64),
                np.mean(pair_valid[territory_mask], dtype=np.float64),
                np.mean(mtt_valid[territory_mask], dtype=np.float64),
                np.mean(mtt_support[territory_mask], dtype=np.float64),
                np.mean(cow_soft_union[territory_mask], dtype=np.float64)
                if cow_prior_valid
                else 0.0,
                np.max(cow_soft_union[territory_mask])
                if cow_prior_valid
                else 0.0,
                np.mean(cow_label[territory_mask] > 0, dtype=np.float64)
                if cow_prior_valid
                else 0.0,
                np.mean(lvo_proximity[territory_mask], dtype=np.float64),
                np.max(lvo_proximity[territory_mask]),
                np.mean(lvo_proximity[territory_mask] >= 0.5, dtype=np.float64),
            ],
            dtype=np.float32,
        )
        meta_valid = np.ones(len(META_FEATURE_NAMES), dtype=np.uint8)
        if not cow_prior_valid:
            for name in [
                'cow_soft_union_mean',
                'cow_soft_union_max',
                'cow_label_overlap_fraction',
            ]:
                meta_valid[META_FEATURE_NAMES.index(name)] = 0
        node_features[node_index, offset:] = meta
        node_feature_valid[node_index, offset:] = meta_valid

    vessel_presence = np.asarray(sidecar['vessel_presence'], dtype=np.uint8)
    atlas_mode = str(row['trial8_atlas_mode'])
    registration_confidence = float(row['trial8_registration_confidence'])
    lvo_present = bool(np.asarray(sidecar['lvo_present']).item())
    case_features = np.asarray(
        [
            1.0,
            registration_confidence,
            float(atlas_mode == 'syn'),
            float(atlas_mode == 'affine'),
            0.0,
            float(cow_prior_valid),
            float(lvo_present),
            float(np.mean(vessel_presence > 0, dtype=np.float64)),
            float(np.mean(perfusion_valid[brain_mask], dtype=np.float64)),
            float(np.mean(pair_valid[brain_mask], dtype=np.float64)),
            float(np.log1p(brain_volume_ml)),
        ],
        dtype=np.float32,
    )

    return {
        'node_features': node_features,
        'node_feature_valid': node_feature_valid,
        'node_valid': node_valid,
        'node_voxels': node_voxels,
        'channel_sample_counts': sample_counts,
        'case_features': case_features,
    }






territory_values = {
    split: [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
    for split in EXPECTED_SPLITS
}
vessel_values = {
    split: [[] for _ in EXPECTED_VESSEL_FEATURE_NAMES]
    for split in EXPECTED_SPLITS
}
case_values = {
    split: [[] for _ in EXPECTED_CASE_FEATURE_NAMES]
    for split in EXPECTED_SPLITS
}

case_rows = []
error_rows = []

for case_index, row in manifest_df.sort_values('subject_id').reset_index(
    drop=True
).iterrows():
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    atlas_mode = str(row['trial8_atlas_mode'])
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    cow_prior_valid = as_bool(row['cow_prior_valid'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    semantic_recomputed = subject_id in recompute_subjects

    try:
        if not sidecar_path.exists():
            raise FileNotFoundError(f'Missing sidecar: {sidecar_path}')
        if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
            raise RuntimeError('Sidecar hash differs from candidate manifest.')
        if not npz_path.exists():
            raise FileNotFoundError(f'Missing source NPZ: {npz_path}')
        if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
            raise RuntimeError('Source NPZ hash differs from frozen manifest.')

        with np.load(sidecar_path, allow_pickle=False) as data:
            sidecar_keys = set(data.files)
            forbidden = sorted(sidecar_keys & FORBIDDEN_SIDECAR_KEYS)
            if forbidden:
                raise RuntimeError(f'Forbidden target arrays in sidecar: {forbidden}')
            sidecar = {name: np.asarray(data[name]) for name in data.files}

        required_sidecar_keys = {
            'subject_id',
            'feature_version',
            'node_labels',
            'node_features',
            'node_feature_valid',
            'node_valid',
            'node_reliability',
            'node_voxels',
            'channel_sample_counts',
            'node_feature_names',
            'case_features',
            'case_feature_names',
            'vessel_features',
            'vessel_feature_valid',
            'vessel_presence',
            'vessel_names',
            'vessel_feature_names',
            'atlas_enabled',
            'atlas_mode',
            'registration_confidence',
            'cow_prior_valid',
            'lvo_present',
            'brain_volume_ml',
            'atlas_valid_voxels',
            'input_npz_sha256',
            'graph_schema_sha256',
            'normalization_fitted',
            'target_arrays_read',
            'study_trainable_parameters',
        }
        missing = sorted(required_sidecar_keys - set(sidecar))
        if missing:
            raise RuntimeError(f'Sidecar missing arrays: {missing}')

        node_features = sidecar['node_features'].astype(np.float32)
        node_feature_valid = sidecar['node_feature_valid'].astype(np.uint8)
        node_valid = sidecar['node_valid'].astype(np.uint8)
        node_reliability = sidecar['node_reliability'].astype(np.float32)
        node_voxels = sidecar['node_voxels'].astype(np.int64)
        sample_counts = sidecar['channel_sample_counts'].astype(np.int64)
        current_case_features = sidecar['case_features'].astype(np.float32)
        current_vessel_features = sidecar['vessel_features'].astype(np.float32)
        vessel_feature_valid = sidecar['vessel_feature_valid'].astype(np.uint8)
        vessel_presence = sidecar['vessel_presence'].astype(np.uint8)

        with np.load(npz_path, allow_pickle=False) as source:
            missing_source = sorted(SOURCE_KEYS_FOR_ALL_CASES - set(source.files))
            if missing_source:
                raise RuntimeError(f'Source NPZ missing arrays: {missing_source}')
            source_subject = scalar_string(source['subject_id'])
            source_lvo_present = bool(np.asarray(source['lvo_present']).item())
            source_vessel_features = np.asarray(
                source['vessel_features_raw'], dtype=np.float32
            )
            source_vessel_presence = np.asarray(
                source['vessel_presence'], dtype=np.uint8
            )
            source_vessel_names = decode_strings(source['vessel_names'])
            source_vessel_feature_names = decode_strings(
                source['vessel_feature_names']
            )

        expected_vessel_features = source_vessel_features.copy()
        expected_vessel_presence = source_vessel_presence.copy()
        if not cow_prior_valid:
            expected_vessel_features.fill(0.0)
            expected_vessel_presence.fill(0)
        expected_node_reliability = np.asarray(
            float(row['trial8_registration_confidence'])
            * np.minimum(
                1.0,
                node_voxels.astype(np.float64)
                / NODE_RELIABILITY_FULL_SUPPORT_VOXELS,
            ),
            dtype=np.float32,
        )

        structural_checks = {
            'subject_matches': scalar_string(sidecar['subject_id']) == subject_id,
            'source_subject_matches': source_subject == subject_id,
            'feature_version_matches': scalar_string(sidecar['feature_version'])
            == FEATURE_VERSION,
            'input_hash_matches': scalar_string(sidecar['input_npz_sha256'])
            == str(row['trial8_npz_sha256']),
            'schema_hash_matches': scalar_string(sidecar['graph_schema_sha256'])
            == graph_schema_sha256,
            'node_labels_match': np.array_equal(
                sidecar['node_labels'], np.arange(1, 31, dtype=np.int16)
            ),
            'node_feature_names_match': decode_strings(
                sidecar['node_feature_names']
            )
            == EXPECTED_NODE_FEATURE_NAMES,
            'case_feature_names_match': decode_strings(
                sidecar['case_feature_names']
            )
            == EXPECTED_CASE_FEATURE_NAMES,
            'vessel_names_match': decode_strings(sidecar['vessel_names'])
            == EXPECTED_VESSEL_NAMES
            == source_vessel_names,
            'vessel_feature_names_match': decode_strings(
                sidecar['vessel_feature_names']
            )
            == EXPECTED_VESSEL_FEATURE_NAMES
            == source_vessel_feature_names,
            'node_feature_shape_matches': node_features.shape
            == (NUM_TERRITORY_NODES, len(EXPECTED_NODE_FEATURE_NAMES)),
            'node_validity_shape_matches': node_feature_valid.shape
            == (NUM_TERRITORY_NODES, len(EXPECTED_NODE_FEATURE_NAMES)),
            'node_vector_shapes_match': node_valid.shape
            == node_reliability.shape
            == node_voxels.shape
            == (NUM_TERRITORY_NODES,),
            'sample_count_shape_matches': sample_counts.shape
            == (NUM_TERRITORY_NODES, len(CONTINUOUS_CHANNELS)),
            'case_feature_shape_matches': current_case_features.shape
            == (len(EXPECTED_CASE_FEATURE_NAMES),),
            'vessel_feature_shapes_match': current_vessel_features.shape
            == vessel_feature_valid.shape
            == (NUM_VESSEL_NODES, len(EXPECTED_VESSEL_FEATURE_NAMES)),
            'vessel_presence_shape_matches': vessel_presence.shape
            == (NUM_VESSEL_NODES,),
            'all_numeric_arrays_finite': all(
                np.isfinite(array).all()
                for array in [
                    node_features,
                    node_reliability,
                    current_case_features,
                    current_vessel_features,
                ]
            ),
            'node_feature_valid_is_binary': binary_array(node_feature_valid),
            'node_valid_is_binary': binary_array(node_valid),
            'vessel_feature_valid_is_binary': binary_array(vessel_feature_valid),
            'vessel_presence_is_binary': binary_array(vessel_presence),
            'invalid_node_features_are_zero': not np.any(
                node_features[node_feature_valid == 0] != 0
            ),
            'invalid_vessel_features_are_zero': not np.any(
                current_vessel_features[vessel_feature_valid == 0] != 0
            ),
            'node_reliability_in_unit_interval': bool(
                (node_reliability >= 0).all() and (node_reliability <= 1).all()
            ),
            'node_reliability_matches_support_and_confidence': np.allclose(
                node_reliability,
                expected_node_reliability,
                atol=1e-7,
                rtol=1e-7,
            ),
            'node_voxel_counts_nonnegative': bool((node_voxels >= 0).all()),
            'sample_counts_nonnegative': bool((sample_counts >= 0).all()),
            'node_valid_matches_voxel_support': np.array_equal(
                node_valid > 0, node_voxels > 0
            ),
            'atlas_enabled_matches_manifest': bool(
                np.asarray(sidecar['atlas_enabled']).item()
            )
            == atlas_enabled,
            'atlas_mode_matches_manifest': scalar_string(sidecar['atlas_mode'])
            == atlas_mode,
            'registration_confidence_matches_manifest': np.isclose(
                float(np.asarray(sidecar['registration_confidence']).item()),
                float(row['trial8_registration_confidence']),
                atol=1e-7,
                rtol=1e-7,
            ),
            'cow_policy_matches_manifest': bool(
                np.asarray(sidecar['cow_prior_valid']).item()
            )
            == cow_prior_valid,
            'lvo_state_matches_source': bool(
                np.asarray(sidecar['lvo_present']).item()
            )
            == source_lvo_present,
            'vessel_presence_matches_source_policy': np.array_equal(
                vessel_presence, expected_vessel_presence
            ),
            'vessel_features_match_source_policy': np.allclose(
                current_vessel_features,
                expected_vessel_features,
                atol=1e-6,
                rtol=1e-6,
            ),
            'case_atlas_enabled_matches': np.isclose(
                current_case_features[0], float(atlas_enabled), atol=1e-7
            ),
            'case_registration_confidence_matches': np.isclose(
                current_case_features[1],
                float(row['trial8_registration_confidence']),
                atol=1e-7,
                rtol=1e-7,
            ),
            'case_mode_one_hot_matches': np.array_equal(
                current_case_features[2:5],
                np.asarray(
                    [
                        float(atlas_mode == 'syn'),
                        float(atlas_mode == 'affine'),
                        float(atlas_mode == 'disabled'),
                    ],
                    dtype=np.float32,
                ),
            ),
            'case_cow_policy_matches': np.isclose(
                current_case_features[5], float(cow_prior_valid), atol=1e-7
            ),
            'case_lvo_state_matches': np.isclose(
                current_case_features[6], float(source_lvo_present), atol=1e-7
            ),
            'case_vessel_fraction_matches': np.isclose(
                current_case_features[7],
                float(np.mean(expected_vessel_presence > 0, dtype=np.float64)),
                atol=1e-7,
                rtol=1e-7,
            ),
            'normalization_not_fit_on_full_dataset': not bool(
                np.asarray(sidecar['normalization_fitted']).item()
            ),
            'target_arrays_not_read': not bool(
                np.asarray(sidecar['target_arrays_read']).item()
            ),
            'study_trainable_parameters_zero': int(
                np.asarray(sidecar['study_trainable_parameters']).item()
            )
            == 0,
            'brain_volume_plausible': 500.0
            <= float(np.asarray(sidecar['brain_volume_ml']).item())
            <= 2200.0,
        }

        if atlas_enabled:
            structural_checks.update({
                'all_30_territory_nodes_valid': int(node_valid.sum()) == 30,
                'all_node_voxel_counts_positive': bool((node_voxels > 0).all()),
                'all_node_reliabilities_positive': bool(
                    (node_reliability > 0).all()
                ),
                'atlas_valid_voxels_positive': int(
                    np.asarray(sidecar['atlas_valid_voxels']).item()
                )
                > 0,
            })
        else:
            structural_checks.update({
                'disabled_node_features_zero': not node_features.any(),
                'disabled_node_feature_valid_zero': not node_feature_valid.any(),
                'disabled_node_valid_zero': not node_valid.any(),
                'disabled_node_reliability_zero': not node_reliability.any(),
                'disabled_node_voxels_zero': not node_voxels.any(),
                'disabled_sample_counts_zero': not sample_counts.any(),
                'disabled_atlas_valid_voxels_zero': int(
                    np.asarray(sidecar['atlas_valid_voxels']).item()
                )
                == 0,
            })

        max_feature_error = 0.0
        max_case_feature_error = 0.0
        semantic_checks = {'semantic_recomputation_not_selected': True}
        if semantic_recomputed:
            recomputed = recompute_subject(row, sidecar)
            semantic_checks = {
                'recomputed_node_valid_matches': np.array_equal(
                    node_valid, recomputed['node_valid']
                ),
                'recomputed_node_voxels_match': np.array_equal(
                    node_voxels, recomputed['node_voxels']
                ),
                'recomputed_sample_counts_match': np.array_equal(
                    sample_counts, recomputed['channel_sample_counts']
                ),
                'recomputed_feature_valid_matches': np.array_equal(
                    node_feature_valid, recomputed['node_feature_valid']
                ),
                'recomputed_node_features_match': np.allclose(
                    node_features,
                    recomputed['node_features'],
                    atol=FEATURE_ATOL,
                    rtol=FEATURE_RTOL,
                ),
            }
            max_feature_error = float(
                np.max(np.abs(node_features - recomputed['node_features']))
            )
            if recomputed['case_features'] is not None:
                semantic_checks['recomputed_case_features_match'] = np.allclose(
                    current_case_features,
                    recomputed['case_features'],
                    atol=FEATURE_ATOL,
                    rtol=FEATURE_RTOL,
                )
                max_case_feature_error = float(
                    np.max(
                        np.abs(
                            current_case_features - recomputed['case_features']
                        )
                    )
                )

        checks = {**structural_checks, **semantic_checks}
        case_passed = bool(all(checks.values()))
        if not case_passed:
            failed = [name for name, passed in checks.items() if not passed]
            raise RuntimeError('Failed checks: ' + ', '.join(failed))

        valid_node_mask = node_valid > 0
        for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
            valid = valid_node_mask & (node_feature_valid[:, feature_index] > 0)
            territory_values[split][feature_index].extend(
                node_features[valid, feature_index].astype(float).tolist()
            )
        valid_vessels = vessel_presence > 0
        for feature_index in range(len(EXPECTED_VESSEL_FEATURE_NAMES)):
            valid = valid_vessels & (vessel_feature_valid[:, feature_index] > 0)
            vessel_values[split][feature_index].extend(
                current_vessel_features[valid, feature_index].astype(float).tolist()
            )
        for feature_index in range(len(EXPECTED_CASE_FEATURE_NAMES)):
            case_values[split][feature_index].append(
                float(current_case_features[feature_index])
            )

        case_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'atlas_mode': atlas_mode,
            'atlas_enabled': atlas_enabled,
            'cow_prior_valid': cow_prior_valid,
            'valid_territory_nodes': int(node_valid.sum()),
            'present_vessels': int(vessel_presence.sum()),
            'valid_node_feature_values': int(node_feature_valid.sum()),
            'valid_vessel_feature_values': int(vessel_feature_valid.sum()),
            'semantic_recomputed': semantic_recomputed,
            'maximum_node_feature_error': max_feature_error,
            'maximum_case_feature_error': max_case_feature_error,
            'sidecar_sha256': str(row['trial8_graph_sidecar_sha256']),
            'target_arrays_read': False,
            'study_trainable_parameters': 0,
            'case_verification_passed': True,
        })

        completed = case_index + 1
        if completed == 1 or completed % 10 == 0 or completed == EXPECTED_SUBJECTS:
            print(
                f'Verified {completed}/{EXPECTED_SUBJECTS}: {subject_id}, '
                f'mode={atlas_mode}, semantic={semantic_recomputed}'
            )

    except Exception as error:
        error_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'error': f'{type(error).__name__}: {error}',
        })
        print(
            f'Failed {case_index + 1}/{EXPECTED_SUBJECTS}: {subject_id}: '
            f'{type(error).__name__}: {error}'
        )

case_df = pd.DataFrame(case_rows)
if len(case_df):
    case_df = case_df.sort_values('subject_id').reset_index(drop=True)
error_df = pd.DataFrame(error_rows, columns=['subject_id', 'dev_split', 'error'])
atomic_write_csv(case_df, CASE_AUDIT_PATH)
atomic_write_csv(error_df, ERROR_AUDIT_PATH)






def distribution_rows(values_by_split, feature_names, feature_group):
    rows = []
    train_arrays = [
        np.asarray(values_by_split['train'][index], dtype=np.float64)
        for index in range(len(feature_names))
    ]
    for feature_index, feature_name in enumerate(feature_names):
        train_values = train_arrays[feature_index]
        train_min = float(train_values.min()) if train_values.size else np.nan
        train_max = float(train_values.max()) if train_values.size else np.nan
        for split in ['train', 'val', 'test']:
            values = np.asarray(
                values_by_split[split][feature_index], dtype=np.float64
            )
            outside_train = (
                int(((values < train_min) | (values > train_max)).sum())
                if values.size and train_values.size
                else 0
            )
            rows.append({
                'feature_group': feature_group,
                'feature_index': feature_index,
                'feature_name': feature_name,
                'dev_split': split,
                'valid_values': int(values.size),
                'mean': float(values.mean()) if values.size else np.nan,
                'std': float(values.std()) if values.size else np.nan,
                'minimum': float(values.min()) if values.size else np.nan,
                'maximum': float(values.max()) if values.size else np.nan,
                'train_minimum': train_min,
                'train_maximum': train_max,
                'outside_train_range_values': outside_train,
                'train_feature_constant': bool(
                    train_values.size > 0
                    and np.isclose(train_values.std(), 0.0, atol=1e-12)
                ),
            })
    return pd.DataFrame(rows)


territory_distribution_df = distribution_rows(
    territory_values, EXPECTED_NODE_FEATURE_NAMES, 'territory'
)
vessel_distribution_df = distribution_rows(
    vessel_values, EXPECTED_VESSEL_FEATURE_NAMES, 'vessel'
)
case_distribution_df = distribution_rows(
    case_values, EXPECTED_CASE_FEATURE_NAMES, 'case'
)

atomic_write_csv(territory_distribution_df, TERRITORY_DISTRIBUTION_PATH)
atomic_write_csv(vessel_distribution_df, VESSEL_DISTRIBUTION_PATH)
atomic_write_csv(case_distribution_df, CASE_DISTRIBUTION_PATH)






all_cases_passed = (
    len(case_df) == EXPECTED_SUBJECTS
    and case_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    and len(error_df) == 0
    and case_df['case_verification_passed'].map(as_bool).all()
)
all_recompute_subjects_covered = (
    all_cases_passed
    and set(
        case_df.loc[case_df['semantic_recomputed'], 'subject_id'].astype(str)
    )
    == recompute_subjects
)
territory_train_support = bool(
    (
        territory_distribution_df.loc[
            territory_distribution_df['dev_split'] == 'train', 'valid_values'
        ]
        > 0
    ).all()
)
vessel_train_support = bool(
    (
        vessel_distribution_df.loc[
            vessel_distribution_df['dev_split'] == 'train', 'valid_values'
        ]
        > 0
    ).all()
)
all_distribution_values_finite = bool(
    all(
        np.isfinite(frame[['mean', 'std', 'minimum', 'maximum']].to_numpy()).all()
        for frame in [
            territory_distribution_df,
            vessel_distribution_df,
            case_distribution_df,
        ]
    )
)

verification_checks = {
    **generation_checks,
    **manifest_checks,
    **schema_conversion_checks,
    'all_topology_checks_passed': bool(all(topology_checks.values())),
    'all_148_sidecars_passed': all_cases_passed,
    'semantic_recomputation_subset_complete': all_recompute_subjects_covered,
    'all_territory_features_have_train_support': territory_train_support,
    'all_vessel_features_have_train_support': vessel_train_support,
    'all_distribution_summaries_finite': all_distribution_values_finite,
    'target_arrays_not_read': True,
    'study_trainable_parameters_are_zero': True,
}
verification_passed = bool(all(verification_checks.values()))

if verification_passed:
    verified_manifest_df = manifest_df.copy()
    verified_manifest_df['trial8_source_graph_schema_path'] = str(
        GRAPH_SCHEMA_PATH
    )
    verified_manifest_df['trial8_source_graph_schema_sha256'] = (
        graph_schema_sha256
    )
    verified_manifest_df['trial8_graph_schema_path'] = str(
        VERIFIED_GRAPH_SCHEMA_PATH
    )
    verified_manifest_df['trial8_graph_schema_sha256'] = (
        verified_graph_schema_sha256
    )
    verified_manifest_df['trial8_graph_schema_pickle_free'] = True
    verified_manifest_df['trial8_graph_verification_version'] = (
        VERIFICATION_VERSION
    )
    verified_manifest_df['trial8_graph_feature_verification_passed'] = True
    verified_manifest_df['trial8_graph_training_scaler_policy'] = (
        'fit_on_train_only_then_freeze'
    )
    verified_manifest_df['trial8_model_training_authorized'] = True
    atomic_write_csv(verified_manifest_df, VERIFIED_MANIFEST_PATH)
    verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
else:
    verified_manifest_sha256 = ''

summary_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'verification_version': VERIFICATION_VERSION,
    'code_revision': CODE_REVISION,
    'verification_passed': verification_passed,
    'checks': verification_checks,
    'subjects_verified': int(len(case_df)),
    'subjects_failed': int(len(error_df)),
    'semantic_recomputation_subjects': sorted(recompute_subjects),
    'semantic_recomputation_subject_count': len(recompute_subjects),
    'maximum_recomputed_node_feature_error': float(
        case_df['maximum_node_feature_error'].max()
    )
    if len(case_df)
    else None,
    'maximum_recomputed_case_feature_error': float(
        case_df['maximum_case_feature_error'].max()
    )
    if len(case_df)
    else None,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH),
    'candidate_manifest_sha256': candidate_manifest_sha256,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH)
    if verification_passed
    else '',
    'verified_manifest_sha256': verified_manifest_sha256,
    'source_graph_schema_path': str(GRAPH_SCHEMA_PATH),
    'source_graph_schema_sha256': graph_schema_sha256,
    'graph_schema_path': str(VERIFIED_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': verified_graph_schema_sha256,
    'graph_schema_pickle_free': True,
    'converted_object_schema_arrays': object_schema_array_names,
    'normalization_policy': 'fit_on_train_only_then_freeze',
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'model_training_authorized': verification_passed,
}
atomic_write_json(summary_record, VERIFICATION_SUMMARY_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'verification_version': VERIFICATION_VERSION,
    'verification_passed': verification_passed,
    'checks': verification_checks,
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH),
    'candidate_manifest_sha256': candidate_manifest_sha256,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH)
    if verification_passed
    else '',
    'verified_manifest_sha256': verified_manifest_sha256,
    'source_graph_schema_path': str(GRAPH_SCHEMA_PATH),
    'source_graph_schema_sha256': graph_schema_sha256,
    'graph_schema_path': str(VERIFIED_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': verified_graph_schema_sha256,
    'graph_schema_pickle_free': True,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
    'model_training_authorized': verification_passed,
}
atomic_write_json(gate_record, VERIFICATION_GATE_PATH)

print('\nVerification checks:')
display(
    pd.DataFrame(
        [
            {'check': name, 'passed': passed}
            for name, passed in verification_checks.items()
        ]
    )
)

if len(case_df):
    print('\nVerified sidecars by atlas mode:')
    display(
        case_df.groupby('atlas_mode', as_index=False).agg(
            subjects=('subject_id', 'count'),
            valid_territory_nodes=('valid_territory_nodes', 'mean'),
            mean_present_vessels=('present_vessels', 'mean'),
            semantic_recomputed=('semantic_recomputed', 'sum'),
            maximum_node_feature_error=('maximum_node_feature_error', 'max'),
        )
    )

print('\nTrain-only support and constant-feature audit:')
display(pd.DataFrame([
    {
        'feature_group': 'territory',
        'features': len(EXPECTED_NODE_FEATURE_NAMES),
        'features_with_train_support': int(
            (
                territory_distribution_df.loc[
                    territory_distribution_df['dev_split'] == 'train',
                    'valid_values',
                ]
                > 0
            ).sum()
        ),
        'constant_train_features': int(
            territory_distribution_df.loc[
                territory_distribution_df['dev_split'] == 'train',
                'train_feature_constant',
            ].sum()
        ),
    },
    {
        'feature_group': 'vessel',
        'features': len(EXPECTED_VESSEL_FEATURE_NAMES),
        'features_with_train_support': int(
            (
                vessel_distribution_df.loc[
                    vessel_distribution_df['dev_split'] == 'train',
                    'valid_values',
                ]
                > 0
            ).sum()
        ),
        'constant_train_features': int(
            vessel_distribution_df.loc[
                vessel_distribution_df['dev_split'] == 'train',
                'train_feature_constant',
            ].sum()
        ),
    },
    {
        'feature_group': 'case',
        'features': len(EXPECTED_CASE_FEATURE_NAMES),
        'features_with_train_support': int(
            (
                case_distribution_df.loc[
                    case_distribution_df['dev_split'] == 'train',
                    'valid_values',
                ]
                > 0
            ).sum()
        ),
        'constant_train_features': int(
            case_distribution_df.loc[
                case_distribution_df['dev_split'] == 'train',
                'train_feature_constant',
            ].sum()
        ),
    },
]))

if not verification_passed:
    if len(error_df):
        print('\nVerification errors:')
        display(error_df)
    failed = [name for name, passed in verification_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 GRAPH SIDECAR VERIFICATION FAILED. Model training remains '
        'locked. Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCULAR GRAPH SIDECAR VERIFICATION PASSED')
print('=' * 80)
print(f'Subjects verified:                  {len(case_df)}')
print(f'Semantic recomputation subjects:    {len(recompute_subjects)}')
print('Target arrays read:                 False')
print('Study trainable parameters:         0 (verification only)')
print('Graph feature verification passed:  True')
print('Model training authorized:          True')
print('Required scaler policy:             fit on train only, then freeze')
print(f'Verified manifest:                  {VERIFIED_MANIFEST_PATH}')
print(f'Verified manifest SHA-256:           {verified_manifest_sha256}')
print(f'Pickle-free graph schema:            {VERIFIED_GRAPH_SCHEMA_PATH}')
print(f'Pickle-free schema SHA-256:          {verified_graph_schema_sha256}')
print(f'Case audit:                         {CASE_AUDIT_PATH}')
print(f'Topology audit:                     {TOPOLOGY_AUDIT_PATH}')
print(f'Verification gate:                  {VERIFICATION_GATE_PATH}')
print('\nNext: build the standalone Trial8 model and train-only scaler preflight.')


In [ ]:
import sys
import os
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
CODE_REVISION = 'r1_standalone_model_scaler_preflight'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = OUTPUT_ROOT / 'architecture_config.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'model_scaler_preflight_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'model_scaler_preflight_audit.csv'

SEED = 20260919
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

atomic_write_npz(
    SCALER_NPZ_PATH,
    territory_feature_names=np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    territory_mean=territory_mean,
    territory_std=territory_std,
    territory_count=territory_count,
    vessel_feature_names=np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    vessel_source_indices=np.asarray(vessel_source_indices, dtype=np.int16),
    vessel_mean=vessel_mean,
    vessel_std=vessel_std,
    vessel_count=vessel_count,
    case_feature_names=np.asarray(MODEL_CASE_FEATURE_NAMES),
    case_source_indices=np.asarray(case_source_indices, dtype=np.int16),
    case_scale_mask=case_scale_mask.astype(np.uint8),
    case_mean=case_mean,
    case_std=case_std,
    case_count=case_count,
    clip_value=np.asarray(SCALER_CLIP, dtype=np.float32),
    fit_split=np.asarray('train'),
    fit_subject_count=np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    fit_subject_hash=np.asarray(scaler_fit_subject_hash),
    target_arrays_read=np.asarray(False, dtype=np.uint8),
)
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(scaler_record, SCALER_JSON_PATH)

print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'image_channels': int(image.shape[0]),
    'patch_size': PATCH_SIZE,
    'backbone_channels': [32, 64, 128, 224, 320],
    'graph_variant': 'two_layer_relation_aware_gated_GATv2',
    'graph_hidden': GRAPH_HIDDEN,
    'graph_heads': GRAPH_HEADS,
    'graph_layers': GRAPH_LAYERS,
    'relation_bases': RELATION_BASES,
    'graph_dropout': GRAPH_DROPOUT,
    'edge_weight_use': 'log_attention_prior',
    'territory_nodes': NUM_TERRITORY_NODES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'conditioning': 'bounded_zero_initialized_multiscale_FiLM',
    'film_bound': FILM_BOUND,
    'graph_to_logit_residual': False,
    'expert_router': False,
    'deep_supervision': True,
    'territory_auxiliary_head': True,
    'scaler_policy': 'fit_on_train_only_then_freeze',
    'expected_total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
    },
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_sha256': scaler_sha256,
    'target_arrays_read': False,
}
atomic_write_json(architecture_config, ARCHITECTURE_JSON_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, PREFLIGHT_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')


In [ ]:
import sys
import os
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v1_target_aware_loss_and_inference'
CODE_REVISION = 'r1_target_aware_training_preflight'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')






print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT')
print('=' * 80)
print('Target access policy: train and validation only; test remains sealed.')

target_rows = []
target_arrays_read_splits = set()
for row in manifest_df[
    manifest_df['dev_split'].isin(['train', 'val'])
].sort_values(['dev_split', 'subject_id']).to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    source_path = Path(str(row['vascufate_v2_npz_path']))
    if not source_path.exists():
        raise FileNotFoundError(f'{subject_id}: frozen NPZ is missing.')
    with np.load(source_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'binary_loss_weight',
            'crop_mask',
            'target_spacing',
            'subject_id',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        spatial_shape = tuple(int(value) for value in target_binary.shape)
        crop_mask = np.asarray(data['crop_mask']) > 0
        binary_loss_weight_value = float(
            np.asarray(data['binary_loss_weight']).item()
        )
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        stored_subject = scalar_string(data['subject_id'])
    target_arrays_read_splits.add(split)
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: stored subject identifier mismatch.')
    if target_soft.shape != target_binary.shape or crop_mask.shape != target_binary.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not set(int(value) for value in np.unique(target_binary)).issubset({0, 1}):
        raise RuntimeError(f'{subject_id}: binary target is not binary.')
    if (
        not np.isfinite(target_soft).all()
        or target_soft.min() < -1e-4
        or target_soft.max() > 1.0001
    ):
        raise RuntimeError(f'{subject_id}: soft target is invalid.')
    if binary_loss_weight_value not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    target_voxels = int(target_binary.sum())
    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
        'spatial_shape': str(spatial_shape),
        'binary_voxels': target_voxels,
        'binary_volume_ml': float(target_voxels * voxel_volume_ml),
        'soft_sum': float(target_soft.sum(dtype=np.float64)),
        'binary_loss_weight': binary_loss_weight_value,
        'target_inside_crop_fraction': 1.0 if target_voxels == 0 else float(
            (target_binary & crop_mask).sum() / target_voxels
        ),
        'all_finite': True,
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_audit_checks = {
    'exactly_102_train_targets_audited': int(
        (target_audit_df['dev_split'] == 'train').sum()
    )
    == 102,
    'exactly_23_validation_targets_audited': int(
        (target_audit_df['dev_split'] == 'val').sum()
    )
    == 23,
    'test_targets_not_opened': target_arrays_read_splits == {'train', 'val'},
    'all_target_arrays_finite': bool(target_audit_df['all_finite'].all()),
    'target_crop_support_fractions_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'train_contains_nonempty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] > 0)
        ).any()
    ),
    'train_contains_empty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] == 0)
        ).any()
    ),
}
if not all(target_audit_checks.values()):
    failed = [
        name for name, passed in target_audit_checks.items() if not passed
    ]
    raise RuntimeError('Target audit failed: ' + ', '.join(failed))

print('\nTarget audit by split:')
display(
    target_audit_df.groupby('dev_split').agg(
        subjects=('subject_id', 'count'),
        empty_targets=('binary_voxels', lambda values: int((values == 0).sum())),
        minimum_volume_ml=('binary_volume_ml', 'min'),
        median_volume_ml=('binary_volume_ml', 'median'),
        maximum_volume_ml=('binary_volume_ml', 'max'),
    ).reset_index()
)
print('\nTarget audit by training lesion bin:')
display(
    target_audit_df[target_audit_df['dev_split'] == 'train']
    .groupby('lesion_volume_bin')
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('binary_volume_ml', 'median'),
    )
    .reset_index()
)






def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)

    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(f'{row["subject_id"]}: atlas labels outside 0..30.')
    return labels, True


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size, spatial_axes):
    source_slices = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source_slices[axis] = spatial_slice
    cropped = array[tuple(source_slices)]
    return center_crop_or_pad(cropped, patch_size, spatial_axes)


def deterministic_patch_center(target, crop_mask, seed, positive_probability=0.75):
    rng = np.random.default_rng(seed)
    positive_coordinates = np.argwhere(target > 0)
    use_positive = (
        len(positive_coordinates) > 0
        and float(rng.random()) < positive_probability
    )
    if use_positive:
        center = positive_coordinates[
            int(rng.integers(0, len(positive_coordinates)))
        ].astype(np.int64)
        jitter = np.asarray([
            int(rng.integers(-16, 17)),
            int(rng.integers(-16, 17)),
            int(rng.integers(-6, 7)),
        ])
        center = center + jitter
        source = 'lesion_jitter'
    else:
        candidates = np.argwhere(crop_mask > 0)
        if len(candidates) == 0:
            center = np.asarray(target.shape) // 2
        else:
            center = candidates[int(rng.integers(0, len(candidates)))]
        source = 'brain_random'
    center = np.clip(center, 0, np.asarray(target.shape) - 1)
    return center, source


manifest_by_subject = {
    str(row['subject_id']): row
    for row in manifest_df.to_dict('records')
}
train_target_df = target_audit_df[target_audit_df['dev_split'] == 'train']
sampling_subjects = []
for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
    matches = train_target_df[
        train_target_df['lesion_volume_bin'] == lesion_bin
    ].sort_values('subject_id')
    if len(matches):
        sampling_subjects.append(str(matches.iloc[0]['subject_id']))
sampling_subjects = list(dict.fromkeys(sampling_subjects))

patch_rows = []
for subject_offset, subject_id in enumerate(sampling_subjects):
    row = manifest_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    for repeat in range(4):
        center, source = deterministic_patch_center(
            target,
            crop_mask,
            SEED + subject_offset * 100 + repeat,
        )
        slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
        patch_target = extract_patch(
            target, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        patch_rows.append({
            'subject_id': subject_id,
            'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
            'repeat': repeat,
            'center_source': source,
            'full_target_voxels': int(target.sum()),
            'patch_target_voxels': int(patch_target.sum()),
            'positive_patch_contains_lesion': (
                source != 'lesion_jitter' or bool(patch_target.any())
            ),
        })

patch_audit_df = pd.DataFrame(patch_rows)
atomic_write_csv(patch_audit_df, PATCH_AUDIT_PATH)
patch_checks = {
    'all_requested_lesion_bins_sampled': len(sampling_subjects) >= 5,
    'four_patches_per_selected_subject': len(patch_audit_df)
    == 4 * len(sampling_subjects),
    'all_lesion_centered_patches_contain_lesion': bool(
        patch_audit_df['positive_patch_contains_lesion'].all()
    ),
    'empty_subject_sampling_supported': bool(
        (patch_audit_df['full_target_voxels'] == 0).any()
    ),
}
if not all(patch_checks.values()):
    failed = [name for name, passed in patch_checks.items() if not passed]
    raise RuntimeError('Patch-sampling checks failed: ' + ', '.join(failed))

print('\nLesion-aware patch audit:')
display(
    patch_audit_df.groupby(
        ['lesion_volume_bin', 'center_source'], dropna=False
    ).agg(
        patches=('subject_id', 'count'),
        minimum_patch_target_voxels=('patch_target_voxels', 'min'),
        maximum_patch_target_voxels=('patch_target_voxels', 'max'),
    ).reset_index()
)






def soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return (1.0 - (2.0 * intersection + epsilon) / (
        denominator + epsilon
    )).mean()


def boundary_weighted_bce(logits, soft_target, binary_target):
    dilated = F.max_pool3d(binary_target, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool3d(
        -binary_target, kernel_size=3, stride=1, padding=1
    )
    boundary = (dilated - eroded).clamp(0.0, 1.0)
    interpolation_uncertainty = 4.0 * soft_target * (1.0 - soft_target)
    weights = 1.0 + 2.0 * boundary + interpolation_uncertainty
    loss = F.binary_cross_entropy_with_logits(
        logits, soft_target, reduction='none'
    )
    return (loss * weights).sum() / weights.sum().clamp_min(1.0)


def focal_tversky_loss(logits, target, alpha=0.30, beta=0.70, gamma=0.75):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return ((1.0 - tversky).clamp_min(0.0) ** gamma)


def segmentation_loss(logits, soft_target, binary_target, hard_loss_weight):
    dice = soft_dice_loss(logits, soft_target)
    boundary_bce = boundary_weighted_bce(logits, soft_target, binary_target)
    hard_tversky_per_case = focal_tversky_loss(logits, binary_target)
    hard_tversky = (
        hard_tversky_per_case * hard_loss_weight
    ).sum() / hard_loss_weight.sum().clamp_min(1.0)
    total = 0.55 * dice + 0.30 * boundary_bce + 0.15 * hard_tversky
    return total, {
        'soft_dice': dice,
        'boundary_bce': boundary_bce,
        'hard_focal_tversky': hard_tversky,
    }


def territory_presence_target(binary_target, region_map):
    targets = []
    for label in range(1, NUM_TERRITORY_NODES + 1):
        targets.append(
            ((binary_target > 0) & (region_map == label)).any().astype(np.float32)
        )
    return np.asarray(targets, dtype=np.float32)


eligible_real_df = train_target_df[
    (train_target_df['binary_voxels'] > 0)
    & (train_target_df['binary_loss_weight'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_real_df = eligible_real_df[
    eligible_real_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
candidate_midpoint = len(eligible_real_df) // 2
candidate_order = sorted(
    range(len(eligible_real_df)),
    key=lambda index: (abs(index - candidate_midpoint), index),
)
real_subject = None
for candidate_index in candidate_order:
    candidate_subject = str(
        eligible_real_df.iloc[candidate_index]['subject_id']
    )
    candidate_row = manifest_by_subject[candidate_subject]
    with np.load(
        Path(str(candidate_row['vascufate_v2_npz_path'])),
        allow_pickle=False,
    ) as candidate_data:
        candidate_target = np.asarray(
            candidate_data['target_binary'], dtype=np.uint8
        )
    candidate_region, _ = load_region_map(
        candidate_row, candidate_target.shape
    )
    if territory_presence_target(
        candidate_target, candidate_region
    ).sum() > 0:
        real_subject = candidate_subject
        break
if real_subject is None:
    raise RuntimeError(
        'No atlas-enabled training lesion overlaps a valid territory.'
    )
real_row = manifest_by_subject[real_subject]
real_graph = load_graph_inputs(real_row)
if file_sha256(Path(str(real_row['vascufate_v2_npz_path']))) != str(
    real_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{real_subject}: real-loss NPZ hash changed.')
with np.load(
    Path(str(real_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    real_image = np.asarray(data['image'], dtype=np.float32)
    real_binary = np.asarray(data['target_binary'], dtype=np.float32)
    real_soft = np.asarray(data['target_soft'], dtype=np.float32)
    real_crop_mask = np.asarray(data['crop_mask']) > 0
    real_binary_weight = float(np.asarray(data['binary_loss_weight']).item())
real_region, real_atlas_enabled = load_region_map(
    real_row, real_binary.shape
)
real_territory_target = territory_presence_target(real_binary, real_region)
real_center, real_center_source = deterministic_patch_center(
    real_binary, real_crop_mask, SEED + 9001, positive_probability=1.0
)
real_slices = patch_slices_from_center(
    real_binary.shape, real_center, PATCH_SIZE
)
real_image_patch = extract_patch(
    real_image, real_slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
)
real_binary_patch = extract_patch(
    real_binary, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_soft_patch = extract_patch(
    real_soft, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_region_patch = extract_patch(
    real_region, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)

real_batch = {
    'image': torch.from_numpy(real_image_patch[None]).to(device),
    'region_map': torch.from_numpy(real_region_patch[None]).to(device),
    'territory_features': torch.from_numpy(
        real_graph['territory_features'][None]
    ).to(device),
    'territory_feature_valid': torch.from_numpy(
        real_graph['territory_feature_valid'][None]
    ).to(device),
    'territory_valid': torch.from_numpy(
        real_graph['territory_valid'][None]
    ).to(device),
    'territory_reliability': torch.from_numpy(
        real_graph['territory_reliability'][None]
    ).to(device),
    'vessel_features': torch.from_numpy(
        real_graph['vessel_features'][None]
    ).to(device),
    'vessel_feature_valid': torch.from_numpy(
        real_graph['vessel_feature_valid'][None]
    ).to(device),
    'vessel_valid': torch.from_numpy(real_graph['vessel_valid'][None]).to(device),
    'case_features': torch.from_numpy(real_graph['case_features'][None]).to(device),
    'atlas_enabled': torch.as_tensor(
        [real_atlas_enabled], dtype=torch.bool, device=device
    ),
}
real_soft_tensor = torch.from_numpy(real_soft_patch[None, None]).to(device)
real_binary_tensor = torch.from_numpy(real_binary_patch[None, None]).to(device)
real_hard_weight = torch.as_tensor(
    [real_binary_weight], dtype=torch.float32, device=device
)
real_territory_target_tensor = torch.from_numpy(
    real_territory_target[None]
).to(device)


set_determinism(SEED + 1)
model = VascuFateNet(
    schema=graph_schema,
    image_channels=real_image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
amp_scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
before_step = model.segmentation_head.weight.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
    enabled=device.type == 'cuda',
):
    real_output = model(**real_batch, force_atlas_dropout=False)
    main_loss, main_parts = segmentation_loss(
        real_output['logits'],
        real_soft_tensor,
        real_binary_tensor,
        real_hard_weight,
    )
    half_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        real_output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        real_hard_weight,
    )
    quarter_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        real_output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        real_hard_weight,
    )
    node_mask = (
        real_batch['territory_valid'].bool()
        & real_output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        real_output['territory_node_logits'],
        real_territory_target_tensor,
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = real_output['territory_node_logits'].sum() * 0.0
    total_real_loss = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )

amp_scaler.scale(total_real_loss).backward()
amp_scaler.unscale_(optimizer)
real_gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0).item()
)
amp_scaler.step(optimizer)
amp_scaler.update()
optimizer.zero_grad(set_to_none=True)
parameter_update = float(
    (model.segmentation_head.weight.detach() - before_step).abs().max().item()
)

loss_values = {
    'subject_id': real_subject,
    'patch_center_source': real_center_source,
    'patch_target_voxels': int(real_binary_patch.sum()),
    'main_loss': float(main_loss.detach().cpu()),
    'soft_dice_loss': float(main_parts['soft_dice'].detach().cpu()),
    'boundary_bce_loss': float(main_parts['boundary_bce'].detach().cpu()),
    'hard_focal_tversky_loss': float(
        main_parts['hard_focal_tversky'].detach().cpu()
    ),
    'half_resolution_loss': float(half_loss.detach().cpu()),
    'quarter_resolution_loss': float(quarter_loss.detach().cpu()),
    'territory_node_loss': float(node_loss.detach().cpu()),
    'total_loss': float(total_real_loss.detach().cpu()),
    'gradient_norm_before_clipping': real_gradient_norm,
    'segmentation_head_maximum_update': parameter_update,
}
atomic_write_json(loss_values, LOSS_AUDIT_PATH)
loss_checks = {
    'real_patch_contains_lesion': int(real_binary_patch.sum()) > 0,
    'all_real_loss_components_finite': all(
        math.isfinite(float(value))
        for key, value in loss_values.items()
        if key.endswith('_loss') or key == 'total_loss'
    ),
    'real_gradient_norm_positive_and_finite': math.isfinite(real_gradient_norm)
    and real_gradient_norm > 0,
    'optimizer_step_changed_parameters': parameter_update > 0,
    'territory_target_has_30_nodes': real_territory_target.shape == (30,),
    'territory_target_contains_positive_node': real_territory_target.sum() > 0,
    'parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
}
if not all(loss_checks.values()):
    failed = [name for name, passed in loss_checks.items() if not passed]
    raise RuntimeError('Real-loss optimizer preflight failed: ' + ', '.join(failed))

print('\nReal-loss disposable optimizer-step audit:')
display(pd.DataFrame([
    {'metric': name, 'value': value} for name, value in loss_values.items()
]))
print('\nParameter count after model reinitialization:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': parameter_count(model)},
    {'component': 'trainable', 'parameters': parameter_count(model, True)},
]))






def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


@torch.no_grad()
def sliding_window_probability(model, row, image, region_map, graph_inputs):
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, PATCH_SIZE)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(PATCH_SIZE)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, PATCH_SIZE)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    model.eval()
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index : location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(
                np.repeat(graph_inputs['territory_features'][None], count, axis=0)
            ).to(device),
            'territory_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'territory_valid': torch.from_numpy(
                np.repeat(graph_inputs['territory_valid'][None], count, axis=0)
            ).to(device),
            'territory_reliability': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_reliability'][None], count, axis=0
                )
            ).to(device),
            'vessel_features': torch.from_numpy(
                np.repeat(graph_inputs['vessel_features'][None], count, axis=0)
            ).to(device),
            'vessel_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['vessel_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'vessel_valid': torch.from_numpy(
                np.repeat(graph_inputs['vessel_valid'][None], count, axis=0)
            ).to(device),
            'case_features': torch.from_numpy(
                np.repeat(graph_inputs['case_features'][None], count, axis=0)
            ).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **inference_batch, force_atlas_dropout=False
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    return probability[original_slices]


eligible_validation_df = target_audit_df[
    (target_audit_df['dev_split'] == 'val')
    & (target_audit_df['binary_voxels'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_validation_df = eligible_validation_df[
    eligible_validation_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
validation_subject = str(
    eligible_validation_df.iloc[len(eligible_validation_df) // 2]['subject_id']
)
validation_row = manifest_by_subject[validation_subject]
validation_graph = load_graph_inputs(validation_row)
if file_sha256(Path(str(validation_row['vascufate_v2_npz_path']))) != str(
    validation_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{validation_subject}: validation NPZ hash changed.')
with np.load(
    Path(str(validation_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    validation_image = np.asarray(data['image'], dtype=np.float32)
    validation_target = np.asarray(data['target_binary'], dtype=np.uint8)
validation_region, _ = load_region_map(
    validation_row, validation_target.shape
)
validation_probability = sliding_window_probability(
    model,
    validation_row,
    validation_image,
    validation_region,
    validation_graph,
)
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    prediction = validation_probability >= threshold
    intersection = int((prediction & (validation_target > 0)).sum())
    predicted = int(prediction.sum())
    truth = int(validation_target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else (2.0 * intersection) / (predicted + truth)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    precision = 1.0 if predicted == 0 and truth == 0 else (
        intersection / max(predicted, 1)
    )
    threshold_rows.append({
        'subject_id': validation_subject,
        'threshold': threshold,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    })
validation_metric_df = pd.DataFrame(threshold_rows)
atomic_write_csv(validation_metric_df, VALIDATION_AUDIT_PATH)
validation_checks = {
    'validation_probability_shape_matches': validation_probability.shape
    == validation_target.shape,
    'validation_probability_finite': bool(
        np.isfinite(validation_probability).all()
    ),
    'validation_probability_in_unit_interval': bool(
        validation_probability.min() >= 0
        and validation_probability.max() <= 1
    ),
    'five_thresholds_evaluated': len(validation_metric_df) == 5,
    'validation_metrics_finite': bool(
        np.isfinite(
            validation_metric_df[['dice', 'precision', 'recall']].to_numpy()
        ).all()
    ),
    'test_targets_still_not_opened': target_arrays_read_splits == {
        'train',
        'val',
    },
}
if not all(validation_checks.values()):
    failed = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Sliding-window validation preflight failed: ' + ', '.join(failed)
    )

print('\nDisposable full-volume validation smoke metrics:')
display(validation_metric_df)






training_policy = {
    'patch_size': list(PATCH_SIZE),
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'maximum_epochs': 100,
    'validation_interval_epochs': 5,
    'early_stopping_patience_validations': 6,
    'optimizer': 'AdamW',
    'learning_rate': 2e-4,
    'weight_decay': 1e-4,
    'scheduler': 'five_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'positive_center_probability': 0.75,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'main_loss_weights': {
        'soft_dice': 0.55,
        'boundary_weighted_bce': 0.30,
        'hard_focal_tversky': 0.15,
    },
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.05,
    'territory_positive_weight': 4.0,
    'validation_inference': '50_percent_overlap_gaussian_sliding_window',
    'threshold_candidates': [
        round(value, 2) for value in np.arange(0.10, 0.61, 0.05)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_training_checks = {
    **frozen_architecture_checks,
    **target_audit_checks,
    **patch_checks,
    **loss_checks,
    **validation_checks,
    'frozen_scaler_hash_matches': scaler_sha256 == EXPECTED_SCALER_SHA256,
    'target_free_model_recheck_passed': preflight_passed,
    'total_parameters_match': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameters_match': parameter_count(model, True)
    == EXPECTED_TOTAL_PARAMETERS,
    'test_split_remains_sealed': 'test' not in target_arrays_read_splits,
}
training_preflight_passed = bool(all(all_training_checks.values()))
atomic_write_csv(
    pd.DataFrame([
        {'check': name, 'passed': passed}
        for name, passed in all_training_checks.items()
    ]),
    PREFLIGHT_AUDIT_PATH,
)
final_gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'training_preflight_passed': training_preflight_passed,
    'full_training_authorized': training_preflight_passed,
    'checks': all_training_checks,
    'training_policy': training_policy,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_sha256': architecture_sha256,
    'total_parameters': parameter_count(model),
    'trainable_parameters': parameter_count(model, True),
    'real_loss_subject': real_subject,
    'validation_smoke_subject': validation_subject,
    'target_arrays_read_splits': sorted(target_arrays_read_splits),
    'test_targets_read': False,
    'study_training_epochs_completed': 0,
}
atomic_write_json(final_gate, PREFLIGHT_GATE_PATH)

print('\nFinal target-aware training preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_training_checks.items()
]))
if not training_preflight_passed:
    failed = [
        name for name, passed in all_training_checks.items() if not passed
    ]
    raise RuntimeError(
        'TRIAL8 TARGET-AWARE TRAINING PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT PASSED')
print('=' * 80)
print('Train targets audited:              102')
print('Validation targets audited:         23')
print('Test targets read:                  False')
print(f'Real-loss subject:                  {real_subject}')
print(f'Validation smoke subject:           {validation_subject}')
print(f'Total parameters:                   {parameter_count(model):,}')
print(f'Trainable parameters:               {parameter_count(model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Frozen scaler SHA-256:              {scaler_sha256}')
print(f'Training preflight gate:            {PREFLIGHT_GATE_PATH}')
print('Study training epochs completed:    0')
print('Full Trial8 training authorized:    True')
print('\nNext: run the separate resume-safe Trial8 training cell.')


In [ ]:
import sys
import os
import gc
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v1_target_aware_loss_and_inference'
CODE_REVISION = 'r1_target_aware_training_preflight'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')






print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT')
print('=' * 80)
print('Target access policy: train and validation only; test remains sealed.')

target_rows = []
target_arrays_read_splits = set()
for row in manifest_df[
    manifest_df['dev_split'].isin(['train', 'val'])
].sort_values(['dev_split', 'subject_id']).to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    source_path = Path(str(row['vascufate_v2_npz_path']))
    if not source_path.exists():
        raise FileNotFoundError(f'{subject_id}: frozen NPZ is missing.')
    with np.load(source_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'binary_loss_weight',
            'crop_mask',
            'target_spacing',
            'subject_id',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        spatial_shape = tuple(int(value) for value in target_binary.shape)
        crop_mask = np.asarray(data['crop_mask']) > 0
        binary_loss_weight_value = float(
            np.asarray(data['binary_loss_weight']).item()
        )
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        stored_subject = scalar_string(data['subject_id'])
    target_arrays_read_splits.add(split)
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: stored subject identifier mismatch.')
    if target_soft.shape != target_binary.shape or crop_mask.shape != target_binary.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not set(int(value) for value in np.unique(target_binary)).issubset({0, 1}):
        raise RuntimeError(f'{subject_id}: binary target is not binary.')
    if (
        not np.isfinite(target_soft).all()
        or target_soft.min() < -1e-4
        or target_soft.max() > 1.0001
    ):
        raise RuntimeError(f'{subject_id}: soft target is invalid.')
    if binary_loss_weight_value not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    target_voxels = int(target_binary.sum())
    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
        'spatial_shape': str(spatial_shape),
        'binary_voxels': target_voxels,
        'binary_volume_ml': float(target_voxels * voxel_volume_ml),
        'soft_sum': float(target_soft.sum(dtype=np.float64)),
        'binary_loss_weight': binary_loss_weight_value,
        'target_inside_crop_fraction': 1.0 if target_voxels == 0 else float(
            (target_binary & crop_mask).sum() / target_voxels
        ),
        'all_finite': True,
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_audit_checks = {
    'exactly_102_train_targets_audited': int(
        (target_audit_df['dev_split'] == 'train').sum()
    )
    == 102,
    'exactly_23_validation_targets_audited': int(
        (target_audit_df['dev_split'] == 'val').sum()
    )
    == 23,
    'test_targets_not_opened': target_arrays_read_splits == {'train', 'val'},
    'all_target_arrays_finite': bool(target_audit_df['all_finite'].all()),
    'target_crop_support_fractions_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'train_contains_nonempty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] > 0)
        ).any()
    ),
    'train_contains_empty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] == 0)
        ).any()
    ),
}
if not all(target_audit_checks.values()):
    failed = [
        name for name, passed in target_audit_checks.items() if not passed
    ]
    raise RuntimeError('Target audit failed: ' + ', '.join(failed))

print('\nTarget audit by split:')
display(
    target_audit_df.groupby('dev_split').agg(
        subjects=('subject_id', 'count'),
        empty_targets=('binary_voxels', lambda values: int((values == 0).sum())),
        minimum_volume_ml=('binary_volume_ml', 'min'),
        median_volume_ml=('binary_volume_ml', 'median'),
        maximum_volume_ml=('binary_volume_ml', 'max'),
    ).reset_index()
)
print('\nTarget audit by training lesion bin:')
display(
    target_audit_df[target_audit_df['dev_split'] == 'train']
    .groupby('lesion_volume_bin')
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('binary_volume_ml', 'median'),
    )
    .reset_index()
)






def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)

    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(f'{row["subject_id"]}: atlas labels outside 0..30.')
    return labels, True


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size, spatial_axes):
    source_slices = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source_slices[axis] = spatial_slice
    cropped = array[tuple(source_slices)]
    return center_crop_or_pad(cropped, patch_size, spatial_axes)


def deterministic_patch_center(target, crop_mask, seed, positive_probability=0.75):
    rng = np.random.default_rng(seed)
    positive_coordinates = np.argwhere(target > 0)
    use_positive = (
        len(positive_coordinates) > 0
        and float(rng.random()) < positive_probability
    )
    if use_positive:
        center = positive_coordinates[
            int(rng.integers(0, len(positive_coordinates)))
        ].astype(np.int64)
        jitter = np.asarray([
            int(rng.integers(-16, 17)),
            int(rng.integers(-16, 17)),
            int(rng.integers(-6, 7)),
        ])
        center = center + jitter
        source = 'lesion_jitter'
    else:
        candidates = np.argwhere(crop_mask > 0)
        if len(candidates) == 0:
            center = np.asarray(target.shape) // 2
        else:
            center = candidates[int(rng.integers(0, len(candidates)))]
        source = 'brain_random'
    center = np.clip(center, 0, np.asarray(target.shape) - 1)
    return center, source


manifest_by_subject = {
    str(row['subject_id']): row
    for row in manifest_df.to_dict('records')
}
train_target_df = target_audit_df[target_audit_df['dev_split'] == 'train']
sampling_subjects = []
for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
    matches = train_target_df[
        train_target_df['lesion_volume_bin'] == lesion_bin
    ].sort_values('subject_id')
    if len(matches):
        sampling_subjects.append(str(matches.iloc[0]['subject_id']))
sampling_subjects = list(dict.fromkeys(sampling_subjects))

patch_rows = []
for subject_offset, subject_id in enumerate(sampling_subjects):
    row = manifest_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    for repeat in range(4):
        center, source = deterministic_patch_center(
            target,
            crop_mask,
            SEED + subject_offset * 100 + repeat,
        )
        slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
        patch_target = extract_patch(
            target, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        patch_rows.append({
            'subject_id': subject_id,
            'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
            'repeat': repeat,
            'center_source': source,
            'full_target_voxels': int(target.sum()),
            'patch_target_voxels': int(patch_target.sum()),
            'positive_patch_contains_lesion': (
                source != 'lesion_jitter' or bool(patch_target.any())
            ),
        })

patch_audit_df = pd.DataFrame(patch_rows)
atomic_write_csv(patch_audit_df, PATCH_AUDIT_PATH)
patch_checks = {
    'all_requested_lesion_bins_sampled': len(sampling_subjects) >= 5,
    'four_patches_per_selected_subject': len(patch_audit_df)
    == 4 * len(sampling_subjects),
    'all_lesion_centered_patches_contain_lesion': bool(
        patch_audit_df['positive_patch_contains_lesion'].all()
    ),
    'empty_subject_sampling_supported': bool(
        (patch_audit_df['full_target_voxels'] == 0).any()
    ),
}
if not all(patch_checks.values()):
    failed = [name for name, passed in patch_checks.items() if not passed]
    raise RuntimeError('Patch-sampling checks failed: ' + ', '.join(failed))

print('\nLesion-aware patch audit:')
display(
    patch_audit_df.groupby(
        ['lesion_volume_bin', 'center_source'], dropna=False
    ).agg(
        patches=('subject_id', 'count'),
        minimum_patch_target_voxels=('patch_target_voxels', 'min'),
        maximum_patch_target_voxels=('patch_target_voxels', 'max'),
    ).reset_index()
)






def soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return (1.0 - (2.0 * intersection + epsilon) / (
        denominator + epsilon
    )).mean()


def boundary_weighted_bce(logits, soft_target, binary_target):
    dilated = F.max_pool3d(binary_target, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool3d(
        -binary_target, kernel_size=3, stride=1, padding=1
    )
    boundary = (dilated - eroded).clamp(0.0, 1.0)
    interpolation_uncertainty = 4.0 * soft_target * (1.0 - soft_target)
    weights = 1.0 + 2.0 * boundary + interpolation_uncertainty
    loss = F.binary_cross_entropy_with_logits(
        logits, soft_target, reduction='none'
    )
    return (loss * weights).sum() / weights.sum().clamp_min(1.0)


def focal_tversky_loss(logits, target, alpha=0.30, beta=0.70, gamma=0.75):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return ((1.0 - tversky).clamp_min(0.0) ** gamma)


def segmentation_loss(logits, soft_target, binary_target, hard_loss_weight):
    dice = soft_dice_loss(logits, soft_target)
    boundary_bce = boundary_weighted_bce(logits, soft_target, binary_target)
    hard_tversky_per_case = focal_tversky_loss(logits, binary_target)
    
    
    hard_loss_weight = hard_loss_weight.reshape(-1, 1)
    hard_tversky = (
        hard_tversky_per_case * hard_loss_weight
    ).sum() / hard_loss_weight.sum().clamp_min(1.0)
    total = 0.55 * dice + 0.30 * boundary_bce + 0.15 * hard_tversky
    return total, {
        'soft_dice': dice,
        'boundary_bce': boundary_bce,
        'hard_focal_tversky': hard_tversky,
    }


def territory_presence_target(binary_target, region_map):
    targets = []
    for label in range(1, NUM_TERRITORY_NODES + 1):
        targets.append(
            ((binary_target > 0) & (region_map == label)).any().astype(np.float32)
        )
    return np.asarray(targets, dtype=np.float32)


eligible_real_df = train_target_df[
    (train_target_df['binary_voxels'] > 0)
    & (train_target_df['binary_loss_weight'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_real_df = eligible_real_df[
    eligible_real_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
candidate_midpoint = len(eligible_real_df) // 2
candidate_order = sorted(
    range(len(eligible_real_df)),
    key=lambda index: (abs(index - candidate_midpoint), index),
)
real_subject = None
for candidate_index in candidate_order:
    candidate_subject = str(
        eligible_real_df.iloc[candidate_index]['subject_id']
    )
    candidate_row = manifest_by_subject[candidate_subject]
    with np.load(
        Path(str(candidate_row['vascufate_v2_npz_path'])),
        allow_pickle=False,
    ) as candidate_data:
        candidate_target = np.asarray(
            candidate_data['target_binary'], dtype=np.uint8
        )
    candidate_region, _ = load_region_map(
        candidate_row, candidate_target.shape
    )
    if territory_presence_target(
        candidate_target, candidate_region
    ).sum() > 0:
        real_subject = candidate_subject
        break
if real_subject is None:
    raise RuntimeError(
        'No atlas-enabled training lesion overlaps a valid territory.'
    )
real_row = manifest_by_subject[real_subject]
real_graph = load_graph_inputs(real_row)
if file_sha256(Path(str(real_row['vascufate_v2_npz_path']))) != str(
    real_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{real_subject}: real-loss NPZ hash changed.')
with np.load(
    Path(str(real_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    real_image = np.asarray(data['image'], dtype=np.float32)
    real_binary = np.asarray(data['target_binary'], dtype=np.float32)
    real_soft = np.asarray(data['target_soft'], dtype=np.float32)
    real_crop_mask = np.asarray(data['crop_mask']) > 0
    real_binary_weight = float(np.asarray(data['binary_loss_weight']).item())
real_region, real_atlas_enabled = load_region_map(
    real_row, real_binary.shape
)
real_territory_target = territory_presence_target(real_binary, real_region)
real_center, real_center_source = deterministic_patch_center(
    real_binary, real_crop_mask, SEED + 9001, positive_probability=1.0
)
real_slices = patch_slices_from_center(
    real_binary.shape, real_center, PATCH_SIZE
)
real_image_patch = extract_patch(
    real_image, real_slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
)
real_binary_patch = extract_patch(
    real_binary, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_soft_patch = extract_patch(
    real_soft, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_region_patch = extract_patch(
    real_region, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)

real_batch = {
    'image': torch.from_numpy(real_image_patch[None]).to(device),
    'region_map': torch.from_numpy(real_region_patch[None]).to(device),
    'territory_features': torch.from_numpy(
        real_graph['territory_features'][None]
    ).to(device),
    'territory_feature_valid': torch.from_numpy(
        real_graph['territory_feature_valid'][None]
    ).to(device),
    'territory_valid': torch.from_numpy(
        real_graph['territory_valid'][None]
    ).to(device),
    'territory_reliability': torch.from_numpy(
        real_graph['territory_reliability'][None]
    ).to(device),
    'vessel_features': torch.from_numpy(
        real_graph['vessel_features'][None]
    ).to(device),
    'vessel_feature_valid': torch.from_numpy(
        real_graph['vessel_feature_valid'][None]
    ).to(device),
    'vessel_valid': torch.from_numpy(real_graph['vessel_valid'][None]).to(device),
    'case_features': torch.from_numpy(real_graph['case_features'][None]).to(device),
    'atlas_enabled': torch.as_tensor(
        [real_atlas_enabled], dtype=torch.bool, device=device
    ),
}
real_soft_tensor = torch.from_numpy(real_soft_patch[None, None]).to(device)
real_binary_tensor = torch.from_numpy(real_binary_patch[None, None]).to(device)
real_hard_weight = torch.as_tensor(
    [real_binary_weight], dtype=torch.float32, device=device
)
real_territory_target_tensor = torch.from_numpy(
    real_territory_target[None]
).to(device)


set_determinism(SEED + 1)
model = VascuFateNet(
    schema=graph_schema,
    image_channels=real_image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
amp_scaler = torch.amp.GradScaler(
    device.type, enabled=device.type == 'cuda'
)
before_step = model.segmentation_head.weight.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
    enabled=device.type == 'cuda',
):
    real_output = model(**real_batch, force_atlas_dropout=False)
    main_loss, main_parts = segmentation_loss(
        real_output['logits'],
        real_soft_tensor,
        real_binary_tensor,
        real_hard_weight,
    )
    half_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        real_output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        real_hard_weight,
    )
    quarter_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        real_output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        real_hard_weight,
    )
    node_mask = (
        real_batch['territory_valid'].bool()
        & real_output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        real_output['territory_node_logits'],
        real_territory_target_tensor,
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = real_output['territory_node_logits'].sum() * 0.0
    total_real_loss = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )

amp_scaler.scale(total_real_loss).backward()
amp_scaler.unscale_(optimizer)
real_gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0).item()
)
amp_scaler.step(optimizer)
amp_scaler.update()
optimizer.zero_grad(set_to_none=True)
parameter_update = float(
    (model.segmentation_head.weight.detach() - before_step).abs().max().item()
)

loss_values = {
    'subject_id': real_subject,
    'patch_center_source': real_center_source,
    'patch_target_voxels': int(real_binary_patch.sum()),
    'main_loss': float(main_loss.detach().cpu()),
    'soft_dice_loss': float(main_parts['soft_dice'].detach().cpu()),
    'boundary_bce_loss': float(main_parts['boundary_bce'].detach().cpu()),
    'hard_focal_tversky_loss': float(
        main_parts['hard_focal_tversky'].detach().cpu()
    ),
    'half_resolution_loss': float(half_loss.detach().cpu()),
    'quarter_resolution_loss': float(quarter_loss.detach().cpu()),
    'territory_node_loss': float(node_loss.detach().cpu()),
    'total_loss': float(total_real_loss.detach().cpu()),
    'gradient_norm_before_clipping': real_gradient_norm,
    'segmentation_head_maximum_update': parameter_update,
}
atomic_write_json(loss_values, LOSS_AUDIT_PATH)
loss_checks = {
    'real_patch_contains_lesion': int(real_binary_patch.sum()) > 0,
    'all_real_loss_components_finite': all(
        math.isfinite(float(value))
        for key, value in loss_values.items()
        if key.endswith('_loss') or key == 'total_loss'
    ),
    'real_gradient_norm_positive_and_finite': math.isfinite(real_gradient_norm)
    and real_gradient_norm > 0,
    'optimizer_step_changed_parameters': parameter_update > 0,
    'territory_target_has_30_nodes': real_territory_target.shape == (30,),
    'territory_target_contains_positive_node': real_territory_target.sum() > 0,
    'parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
}
if not all(loss_checks.values()):
    failed = [name for name, passed in loss_checks.items() if not passed]
    raise RuntimeError('Real-loss optimizer preflight failed: ' + ', '.join(failed))

print('\nReal-loss disposable optimizer-step audit:')
display(pd.DataFrame([
    {'metric': name, 'value': value} for name, value in loss_values.items()
]))
print('\nParameter count after model reinitialization:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': parameter_count(model)},
    {'component': 'trainable', 'parameters': parameter_count(model, True)},
]))






def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


@torch.no_grad()
def sliding_window_probability(model, row, image, region_map, graph_inputs):
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, PATCH_SIZE)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(PATCH_SIZE)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, PATCH_SIZE)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    model.eval()
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index : location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(
                np.repeat(graph_inputs['territory_features'][None], count, axis=0)
            ).to(device),
            'territory_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'territory_valid': torch.from_numpy(
                np.repeat(graph_inputs['territory_valid'][None], count, axis=0)
            ).to(device),
            'territory_reliability': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_reliability'][None], count, axis=0
                )
            ).to(device),
            'vessel_features': torch.from_numpy(
                np.repeat(graph_inputs['vessel_features'][None], count, axis=0)
            ).to(device),
            'vessel_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['vessel_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'vessel_valid': torch.from_numpy(
                np.repeat(graph_inputs['vessel_valid'][None], count, axis=0)
            ).to(device),
            'case_features': torch.from_numpy(
                np.repeat(graph_inputs['case_features'][None], count, axis=0)
            ).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **inference_batch, force_atlas_dropout=False
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    return probability[original_slices]


eligible_validation_df = target_audit_df[
    (target_audit_df['dev_split'] == 'val')
    & (target_audit_df['binary_voxels'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_validation_df = eligible_validation_df[
    eligible_validation_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
validation_subject = str(
    eligible_validation_df.iloc[len(eligible_validation_df) // 2]['subject_id']
)
validation_row = manifest_by_subject[validation_subject]
validation_graph = load_graph_inputs(validation_row)
if file_sha256(Path(str(validation_row['vascufate_v2_npz_path']))) != str(
    validation_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{validation_subject}: validation NPZ hash changed.')
with np.load(
    Path(str(validation_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    validation_image = np.asarray(data['image'], dtype=np.float32)
    validation_target = np.asarray(data['target_binary'], dtype=np.uint8)
validation_region, _ = load_region_map(
    validation_row, validation_target.shape
)
validation_probability = sliding_window_probability(
    model,
    validation_row,
    validation_image,
    validation_region,
    validation_graph,
)
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    prediction = validation_probability >= threshold
    intersection = int((prediction & (validation_target > 0)).sum())
    predicted = int(prediction.sum())
    truth = int(validation_target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else (2.0 * intersection) / (predicted + truth)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    precision = 1.0 if predicted == 0 and truth == 0 else (
        intersection / max(predicted, 1)
    )
    threshold_rows.append({
        'subject_id': validation_subject,
        'threshold': threshold,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    })
validation_metric_df = pd.DataFrame(threshold_rows)
atomic_write_csv(validation_metric_df, VALIDATION_AUDIT_PATH)
validation_checks = {
    'validation_probability_shape_matches': validation_probability.shape
    == validation_target.shape,
    'validation_probability_finite': bool(
        np.isfinite(validation_probability).all()
    ),
    'validation_probability_in_unit_interval': bool(
        validation_probability.min() >= 0
        and validation_probability.max() <= 1
    ),
    'five_thresholds_evaluated': len(validation_metric_df) == 5,
    'validation_metrics_finite': bool(
        np.isfinite(
            validation_metric_df[['dice', 'precision', 'recall']].to_numpy()
        ).all()
    ),
    'test_targets_still_not_opened': target_arrays_read_splits == {
        'train',
        'val',
    },
}
if not all(validation_checks.values()):
    failed = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Sliding-window validation preflight failed: ' + ', '.join(failed)
    )

print('\nDisposable full-volume validation smoke metrics:')
display(validation_metric_df)






training_policy = {
    'patch_size': list(PATCH_SIZE),
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'maximum_epochs': 100,
    'validation_interval_epochs': 5,
    'early_stopping_patience_validations': 6,
    'optimizer': 'AdamW',
    'learning_rate': 2e-4,
    'weight_decay': 1e-4,
    'scheduler': 'five_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'positive_center_probability': 0.75,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'main_loss_weights': {
        'soft_dice': 0.55,
        'boundary_weighted_bce': 0.30,
        'hard_focal_tversky': 0.15,
    },
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.05,
    'territory_positive_weight': 4.0,
    'validation_inference': '50_percent_overlap_gaussian_sliding_window',
    'threshold_candidates': [
        round(value, 2) for value in np.arange(0.10, 0.61, 0.05)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_training_checks = {
    **frozen_architecture_checks,
    **target_audit_checks,
    **patch_checks,
    **loss_checks,
    **validation_checks,
    'frozen_scaler_hash_matches': scaler_sha256 == EXPECTED_SCALER_SHA256,
    'target_free_model_recheck_passed': preflight_passed,
    'total_parameters_match': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameters_match': parameter_count(model, True)
    == EXPECTED_TOTAL_PARAMETERS,
    'test_split_remains_sealed': 'test' not in target_arrays_read_splits,
}
training_preflight_passed = bool(all(all_training_checks.values()))
atomic_write_csv(
    pd.DataFrame([
        {'check': name, 'passed': passed}
        for name, passed in all_training_checks.items()
    ]),
    PREFLIGHT_AUDIT_PATH,
)
final_gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'training_preflight_passed': training_preflight_passed,
    'full_training_authorized': training_preflight_passed,
    'checks': all_training_checks,
    'training_policy': training_policy,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_sha256': architecture_sha256,
    'total_parameters': parameter_count(model),
    'trainable_parameters': parameter_count(model, True),
    'real_loss_subject': real_subject,
    'validation_smoke_subject': validation_subject,
    'target_arrays_read_splits': sorted(target_arrays_read_splits),
    'test_targets_read': False,
    'study_training_epochs_completed': 0,
}
atomic_write_json(final_gate, PREFLIGHT_GATE_PATH)

print('\nFinal target-aware training preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_training_checks.items()
]))
if not training_preflight_passed:
    failed = [
        name for name, passed in all_training_checks.items() if not passed
    ]
    raise RuntimeError(
        'TRIAL8 TARGET-AWARE TRAINING PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT PASSED')
print('=' * 80)
print('Train targets audited:              102')
print('Validation targets audited:         23')
print('Test targets read:                  False')
print(f'Real-loss subject:                  {real_subject}')
print(f'Validation smoke subject:           {validation_subject}')
print(f'Total parameters:                   {parameter_count(model):,}')
print(f'Trainable parameters:               {parameter_count(model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Frozen scaler SHA-256:              {scaler_sha256}')
print(f'Training preflight gate:            {PREFLIGHT_GATE_PATH}')
print('Study training epochs completed:    0')
print('Full Trial8 training authorized:    True')
print('\nTarget-aware gate passed; starting authorized training below.')








del model, optimizer, amp_scaler, real_output, real_batch
del real_soft_tensor, real_binary_tensor, real_territory_target_tensor
del normal_output, dropped_output, gradient_output, batch
del smoke_loss, total_real_loss, main_loss, half_loss, quarter_loss
del node_loss, node_bce, before_step
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TRAINING_VERSION = 'v1_ragatv2_film_boundary_region'
TRAINING_ROOT = (
    PROJECT_ROOT
    / 'checkpoints'
    / 'trial8_vascufate_net_v1'
    / TRAINING_VERSION
)
LAST_CHECKPOINT_PATH = TRAINING_ROOT / 'last_checkpoint.pt'
BEST_CHECKPOINT_PATH = TRAINING_ROOT / 'best_checkpoint.pt'
HISTORY_PATH = TRAINING_ROOT / 'training_history.csv'
VALIDATION_ROOT = TRAINING_ROOT / 'validation'
TRAINING_CONFIG_PATH = TRAINING_ROOT / 'training_config.json'
TRAINING_STATE_PATH = TRAINING_ROOT / 'training_state.json'

assert_project_path_is_safe(TRAINING_ROOT)
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

final_training_gate = json.loads(
    PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if final_training_gate.get('training_preflight_passed') is not True:
    raise RuntimeError('Target-aware training preflight is not passed.')
if final_training_gate.get('full_training_authorized') is not True:
    raise RuntimeError('Full Trial8 training is not authorized.')
if final_training_gate.get('test_targets_read') is not False:
    raise RuntimeError('Training preflight reports test-target access.')
if final_training_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Training gate parameter count changed.')
if final_training_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Training gate scaler hash changed.')

policy = final_training_gate['training_policy']
BATCH_SIZE = int(policy['batch_size'])
ACCUMULATION_STEPS = int(policy['gradient_accumulation_steps'])
MAX_EPOCHS = int(policy['maximum_epochs'])
VALIDATION_INTERVAL = int(policy['validation_interval_epochs'])
PATIENCE_VALIDATIONS = int(policy['early_stopping_patience_validations'])
BASE_LEARNING_RATE = float(policy['learning_rate'])
WEIGHT_DECAY = float(policy['weight_decay'])
GRADIENT_CLIP_NORM = float(policy['gradient_clip_norm'])
POSITIVE_CENTER_PROBABILITY = float(policy['positive_center_probability'])
THRESHOLDS = [float(value) for value in policy['threshold_candidates']]
NUM_WORKERS = 2
IMPROVEMENT_EPSILON = 1e-4

if tuple(policy['patch_size']) != PATCH_SIZE:
    raise RuntimeError('Frozen training patch size changed.')
if int(policy['effective_batch_size']) != BATCH_SIZE * ACCUMULATION_STEPS:
    raise RuntimeError('Frozen effective batch size is inconsistent.')


def atomic_torch_save(record, path):
    path = Path(path)
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


def augment_training_image(image, rng):
    augmented = image.astype(np.float32, copy=True)
    intensity_channels = min(13, augmented.shape[0])
    gains = rng.uniform(0.95, 1.05, size=(intensity_channels, 1, 1, 1))
    offsets = rng.uniform(-0.02, 0.02, size=(intensity_channels, 1, 1, 1))
    augmented[:intensity_channels] = (
        augmented[:intensity_channels] * gains + offsets
    )
    if float(rng.random()) < 0.50:
        noise_scale = float(rng.uniform(0.0, 0.015))
        noise = rng.normal(
            0.0,
            noise_scale,
            size=augmented[:intensity_channels].shape,
        ).astype(np.float32)
        augmented[:intensity_channels] += noise
    non_delta_end = min(9, intensity_channels)
    augmented[:non_delta_end] = np.clip(
        augmented[:non_delta_end], 0.0, 1.0
    )
    if intensity_channels > 9:
        augmented[9:intensity_channels] = np.clip(
            augmented[9:intensity_channels], -1.0, 1.0
        )
    return augmented


class Trial8PatchDataset(torch.utils.data.Dataset):
    def __init__(self, rows, base_seed):
        self.rows = list(rows)
        self.base_seed = int(base_seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        subject_id = str(row['subject_id'])
        rng = np.random.default_rng(
            self.base_seed + self.epoch * 100_003 + index * 997
        )
        with np.load(
            Path(str(row['vascufate_v2_npz_path'])),
            allow_pickle=False,
        ) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target_binary = np.asarray(
                data['target_binary'], dtype=np.float32
            )
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            crop_mask = np.asarray(data['crop_mask']) > 0
            binary_loss_weight = float(
                np.asarray(data['binary_loss_weight']).item()
            )
        region_map, atlas_enabled = load_region_map(
            row, target_binary.shape
        )
        graph = load_graph_inputs(row)
        territory_target = territory_presence_target(
            target_binary, region_map
        )
        center, center_source = deterministic_patch_center(
            target_binary,
            crop_mask,
            self.base_seed + self.epoch * 100_003 + index * 997,
            positive_probability=POSITIVE_CENTER_PROBABILITY,
        )
        slices = patch_slices_from_center(
            target_binary.shape, center, PATCH_SIZE
        )
        image_patch = extract_patch(
            image, slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
        )
        binary_patch = extract_patch(
            target_binary, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        soft_patch = extract_patch(
            target_soft, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        region_patch = extract_patch(
            region_map, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        image_patch = augment_training_image(image_patch, rng)
        return {
            'subject_id': subject_id,
            'center_source': center_source,
            'image': torch.from_numpy(image_patch),
            'region_map': torch.from_numpy(region_patch),
            'target_binary': torch.from_numpy(binary_patch[None]),
            'target_soft': torch.from_numpy(soft_patch[None]),
            'binary_loss_weight': torch.as_tensor(
                binary_loss_weight, dtype=torch.float32
            ),
            'territory_target': torch.from_numpy(territory_target),
            'territory_features': torch.from_numpy(
                graph['territory_features']
            ),
            'territory_feature_valid': torch.from_numpy(
                graph['territory_feature_valid']
            ),
            'territory_valid': torch.from_numpy(graph['territory_valid']),
            'territory_reliability': torch.from_numpy(
                graph['territory_reliability']
            ),
            'vessel_features': torch.from_numpy(graph['vessel_features']),
            'vessel_feature_valid': torch.from_numpy(
                graph['vessel_feature_valid']
            ),
            'vessel_valid': torch.from_numpy(graph['vessel_valid']),
            'case_features': torch.from_numpy(graph['case_features']),
            'atlas_enabled': torch.as_tensor(
                atlas_enabled, dtype=torch.bool
            ),
        }


def to_device(batch, device):
    return {
        key: value.to(device, non_blocking=True)
        if torch.is_tensor(value)
        else value
        for key, value in batch.items()
    }


def compute_training_loss(model, batch):
    model_inputs = {
        key: batch[key]
        for key in [
            'image',
            'region_map',
            'territory_features',
            'territory_feature_valid',
            'territory_valid',
            'territory_reliability',
            'vessel_features',
            'vessel_feature_valid',
            'vessel_valid',
            'case_features',
            'atlas_enabled',
        ]
    }
    output = model(**model_inputs)
    main_loss, main_parts = segmentation_loss(
        output['logits'],
        batch['target_soft'],
        batch['target_binary'],
        batch['binary_loss_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        batch['binary_loss_weight'],
    )
    quarter_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        batch['binary_loss_weight'],
    )
    node_mask = (
        batch['territory_valid'].bool()
        & output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        output['territory_node_logits'],
        batch['territory_target'],
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=batch['image'].device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = output['territory_node_logits'].sum() * 0.0
    total = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )
    parts = {
        'total_loss': total,
        'main_loss': main_loss,
        'soft_dice_loss': main_parts['soft_dice'],
        'boundary_bce_loss': main_parts['boundary_bce'],
        'hard_focal_tversky_loss': main_parts['hard_focal_tversky'],
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
    }
    return total, parts


def epoch_loader(dataset, epoch):
    dataset.set_epoch(epoch)
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda',
        persistent_workers=False,
        drop_last=False,
        generator=generator,
    )


def subject_metrics(probability, target, threshold):
    prediction = probability >= threshold
    target = target > 0
    intersection = int((prediction & target).sum())
    predicted = int(prediction.sum())
    truth = int(target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else 2.0 * intersection / (predicted + truth)
    )
    precision = (
        1.0
        if predicted == 0 and truth == 0
        else intersection / max(predicted, 1)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    return {
        'dice': float(dice),
        'precision': float(precision),
        'recall': float(recall),
        'intersection': intersection,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    }


@torch.no_grad()
def validate_all_subjects(model, validation_rows, epoch):
    threshold_rows = []
    model.eval()
    for case_index, row in enumerate(validation_rows, start=1):
        subject_id = str(row['subject_id'])
        with np.load(
            Path(str(row['vascufate_v2_npz_path'])),
            allow_pickle=False,
        ) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target = np.asarray(data['target_binary'], dtype=np.uint8)
            spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        region_map, _ = load_region_map(row, target.shape)
        graph_inputs = load_graph_inputs(row)
        probability = sliding_window_probability(
            model, row, image, region_map, graph_inputs
        )
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        for threshold in THRESHOLDS:
            metrics = subject_metrics(probability, target, threshold)
            threshold_rows.append({
                'epoch': epoch,
                'subject_id': subject_id,
                'threshold': threshold,
                **metrics,
                'predicted_volume_ml': (
                    metrics['predicted_voxels'] * voxel_volume_ml
                ),
                'target_volume_ml': metrics['target_voxels'] * voxel_volume_ml,
                'absolute_volume_error_ml': abs(
                    metrics['predicted_voxels'] - metrics['target_voxels']
                )
                * voxel_volume_ml,
            })
        if case_index == 1 or case_index % 5 == 0 or case_index == len(validation_rows):
            print(
                f'  Validated {case_index}/{len(validation_rows)}: '
                f'{subject_id}'
            )
    case_frame = pd.DataFrame(threshold_rows)
    summary = (
        case_frame.groupby('threshold')
        .agg(
            mean_subject_dice=('dice', 'mean'),
            median_subject_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_error_ml=('absolute_volume_error_ml', 'mean'),
        )
        .reset_index()
    )
    global_rows = []
    for threshold, group in case_frame.groupby('threshold'):
        intersection = int(group['intersection'].sum())
        predicted = int(group['predicted_voxels'].sum())
        truth = int(group['target_voxels'].sum())
        global_rows.append({
            'threshold': float(threshold),
            'global_dice': (
                1.0
                if predicted + truth == 0
                else 2.0 * intersection / (predicted + truth)
            ),
        })
    summary = summary.merge(pd.DataFrame(global_rows), on='threshold')
    summary = summary.sort_values(
        ['mean_subject_dice', 'mean_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = summary.iloc[0].to_dict()
    epoch_directory = VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_directory.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(case_frame, epoch_directory / 'validation_cases.csv')
    atomic_write_csv(summary, epoch_directory / 'threshold_summary.csv')
    atomic_write_json(
        {
            'epoch': epoch,
            'selected_threshold': float(selected['threshold']),
            'mean_subject_dice': float(selected['mean_subject_dice']),
            'median_subject_dice': float(selected['median_subject_dice']),
            'global_dice': float(selected['global_dice']),
            'mean_precision': float(selected['mean_precision']),
            'mean_recall': float(selected['mean_recall']),
            'test_targets_read': False,
        },
        epoch_directory / 'validation_summary.json',
    )
    return selected


train_rows = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .to_dict('records')
)
validation_rows = (
    manifest_df[manifest_df['dev_split'] == 'val']
    .sort_values('subject_id')
    .to_dict('records')
)
if len(train_rows) != 102 or len(validation_rows) != 23:
    raise RuntimeError('Training/validation split count changed.')

training_dataset = Trial8PatchDataset(train_rows, SEED + 10_000)
updates_per_epoch = math.ceil(
    math.ceil(len(training_dataset) / BATCH_SIZE) / ACCUMULATION_STEPS
)
total_update_steps = MAX_EPOCHS * updates_per_epoch
warmup_steps = 5 * updates_per_epoch


def learning_rate_multiplier(step):
    if step < warmup_steps:
        return float(step + 1) / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(
        total_update_steps - warmup_steps, 1
    )
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


set_determinism(SEED + 2)
training_model = VascuFateNet(
    schema=graph_schema,
    image_channels=15,
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
if parameter_count(training_model) != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Fresh training model parameter count changed.')

training_optimizer = torch.optim.AdamW(
    training_model.parameters(),
    lr=BASE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
training_scheduler = torch.optim.lr_scheduler.LambdaLR(
    training_optimizer,
    lr_lambda=learning_rate_multiplier,
)
training_amp_scaler = torch.amp.GradScaler(
    device.type, enabled=device.type == 'cuda'
)

start_epoch = 1
global_update_step = 0
best_mean_dice = -1.0
best_epoch = 0
best_threshold = 0.20
validations_without_improvement = 0
history_rows = []

if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    if checkpoint.get('training_version') != TRAINING_VERSION:
        raise RuntimeError('Existing checkpoint training version mismatch.')
    if checkpoint.get('verified_manifest_sha256') != verified_manifest_sha256:
        raise RuntimeError('Existing checkpoint manifest hash mismatch.')
    if checkpoint.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
        raise RuntimeError('Existing checkpoint scaler hash mismatch.')
    if checkpoint.get('graph_schema_sha256') != safe_schema_sha256:
        raise RuntimeError('Existing checkpoint graph-schema hash mismatch.')
    if checkpoint.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
        raise RuntimeError('Existing checkpoint parameter count mismatch.')
    training_model.load_state_dict(checkpoint['model_state_dict'])
    training_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    training_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    training_amp_scaler.load_state_dict(checkpoint['amp_scaler_state_dict'])
    restore_rng_state(checkpoint['rng_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    global_update_step = int(checkpoint['global_update_step'])
    best_mean_dice = float(checkpoint['best_mean_dice'])
    best_epoch = int(checkpoint['best_epoch'])
    best_threshold = float(checkpoint['best_threshold'])
    validations_without_improvement = int(
        checkpoint['validations_without_improvement']
    )
    history_rows = list(checkpoint.get('history_rows', []))
    print(
        f'Resuming Trial8 after epoch {start_epoch - 1}; '
        f'best Dice={best_mean_dice:.4f} at epoch {best_epoch}.'
    )
else:
    print('Starting Trial8 from a fresh deterministic initialization.')

training_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_version': TRAINING_VERSION,
    'training_policy': policy,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_sha256': EXPECTED_SCALER_SHA256,
    'graph_schema_sha256': safe_schema_sha256,
    'architecture_sha256': architecture_sha256,
    'total_parameters': parameter_count(training_model),
    'trainable_parameters': parameter_count(training_model, True),
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
    },
    'train_subjects': 102,
    'validation_subjects': 23,
    'test_subjects_accessed': 0,
}
if TRAINING_CONFIG_PATH.exists():
    existing_config = json.loads(
        TRAINING_CONFIG_PATH.read_text(encoding='utf-8')
    )
    immutable_keys = [
        'training_version',
        'verified_manifest_sha256',
        'scaler_sha256',
        'graph_schema_sha256',
        'architecture_sha256',
        'total_parameters',
        'training_policy',
    ]
    if any(
        existing_config.get(key) != training_config.get(key)
        for key in immutable_keys
    ):
        raise RuntimeError('Existing training configuration changed.')
else:
    atomic_write_json(training_config, TRAINING_CONFIG_PATH)

print('\n' + '=' * 80)
print('TRIAL8 RESUME-SAFE TRAINING START')
print('=' * 80)
print(f'Device:                             {device}')
if device.type == 'cuda':
    print(f'GPU:                                {torch.cuda.get_device_name(0)}')
print(f'Train subjects:                     {len(train_rows)}')
print(f'Validation subjects:                {len(validation_rows)}')
print('Test subjects accessed:             0')
print(f'Patch size:                         {PATCH_SIZE}')
print(f'Batch / accumulation / effective:   {BATCH_SIZE} / {ACCUMULATION_STEPS} / {BATCH_SIZE * ACCUMULATION_STEPS}')
print(f'Updates per epoch:                  {updates_per_epoch}')
print(f'Maximum epochs:                     {MAX_EPOCHS}')
print(f'Total parameters:                   {parameter_count(training_model):,}')
print(f'Trainable parameters:               {parameter_count(training_model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Checkpoint directory:               {TRAINING_ROOT}')

training_already_complete = (
    start_epoch > MAX_EPOCHS
    or validations_without_improvement >= PATIENCE_VALIDATIONS
)
stop_reason = (
    'already_complete_on_resume'
    if training_already_complete
    else 'maximum_epochs_reached'
)
if training_already_complete:
    print(
        'The latest checkpoint already satisfies a terminal stopping '
        'condition; no additional epochs will run.'
    )

for epoch in range(
    start_epoch,
    start_epoch if training_already_complete else MAX_EPOCHS + 1,
):
    epoch_start = datetime.now(timezone.utc)
    loader = epoch_loader(training_dataset, epoch)
    training_model.train()
    training_optimizer.zero_grad(set_to_none=True)
    running = {
        'total_loss': 0.0,
        'main_loss': 0.0,
        'soft_dice_loss': 0.0,
        'boundary_bce_loss': 0.0,
        'hard_focal_tversky_loss': 0.0,
        'half_loss': 0.0,
        'quarter_loss': 0.0,
        'node_loss': 0.0,
        'atlas_retained_fraction': 0.0,
    }
    processed_batches = 0
    epoch_gradient_norms = []
    for batch_index, cpu_batch in enumerate(loader, start=1):
        batch = to_device(cpu_batch, device)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            total_loss, parts = compute_training_loss(training_model, batch)
            accumulation_group_start = (
                (batch_index - 1) // ACCUMULATION_STEPS
            ) * ACCUMULATION_STEPS
            accumulation_group_size = min(
                ACCUMULATION_STEPS,
                len(loader) - accumulation_group_start,
            )
            scaled_loss = total_loss / accumulation_group_size
        training_amp_scaler.scale(scaled_loss).backward()
        for name in running:
            running[name] += float(parts[name].detach().cpu())
        processed_batches += 1
        should_update = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(loader)
        )
        if should_update:
            training_amp_scaler.unscale_(training_optimizer)
            gradient_norm = float(
                torch.nn.utils.clip_grad_norm_(
                    training_model.parameters(), GRADIENT_CLIP_NORM
                ).item()
            )
            if not math.isfinite(gradient_norm):
                raise RuntimeError(
                    f'Non-finite gradient norm at epoch {epoch}.'
                )
            epoch_gradient_norms.append(gradient_norm)
            training_amp_scaler.step(training_optimizer)
            training_amp_scaler.update()
            training_optimizer.zero_grad(set_to_none=True)
            training_scheduler.step()
            global_update_step += 1
    epoch_means = {
        name: value / max(processed_batches, 1)
        for name, value in running.items()
    }
    if not all(math.isfinite(value) for value in epoch_means.values()):
        raise RuntimeError(f'Non-finite training loss at epoch {epoch}.')

    validation_result = None
    improved = False
    should_validate = (
        epoch % VALIDATION_INTERVAL == 0 or epoch == MAX_EPOCHS
    )
    if should_validate:
        print(f'Running full validation at epoch {epoch}...')
        validation_result = validate_all_subjects(
            training_model, validation_rows, epoch
        )
        current_dice = float(validation_result['mean_subject_dice'])
        current_threshold = float(validation_result['threshold'])
        if current_dice > best_mean_dice + IMPROVEMENT_EPSILON:
            improved = True
            best_mean_dice = current_dice
            best_epoch = epoch
            best_threshold = current_threshold
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1

    epoch_seconds = (
        datetime.now(timezone.utc) - epoch_start
    ).total_seconds()
    history_row = {
        'epoch': epoch,
        'global_update_step': global_update_step,
        'learning_rate': float(training_optimizer.param_groups[0]['lr']),
        **epoch_means,
        'mean_gradient_norm': float(np.mean(epoch_gradient_norms)),
        'maximum_gradient_norm': float(np.max(epoch_gradient_norms)),
        'validation_performed': should_validate,
        'validation_threshold': (
            float(validation_result['threshold'])
            if validation_result is not None
            else np.nan
        ),
        'validation_mean_subject_dice': (
            float(validation_result['mean_subject_dice'])
            if validation_result is not None
            else np.nan
        ),
        'validation_global_dice': (
            float(validation_result['global_dice'])
            if validation_result is not None
            else np.nan
        ),
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'improved': improved,
        'epoch_seconds': epoch_seconds,
        'test_targets_read': False,
    }
    history_rows.append(to_builtin(history_row))
    atomic_write_csv(pd.DataFrame(history_rows), HISTORY_PATH)

    checkpoint_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'epoch': epoch,
        'global_update_step': global_update_step,
        'model_state_dict': training_model.state_dict(),
        'optimizer_state_dict': training_optimizer.state_dict(),
        'scheduler_state_dict': training_scheduler.state_dict(),
        'amp_scaler_state_dict': training_amp_scaler.state_dict(),
        'rng_state': capture_rng_state(),
        'best_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'history_rows': history_rows,
        'verified_manifest_sha256': verified_manifest_sha256,
        'scaler_sha256': EXPECTED_SCALER_SHA256,
        'graph_schema_sha256': safe_schema_sha256,
        'architecture_sha256': architecture_sha256,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'test_targets_read': False,
    }
    atomic_torch_save(checkpoint_record, LAST_CHECKPOINT_PATH)
    if improved:
        atomic_torch_save(checkpoint_record, BEST_CHECKPOINT_PATH)

    print(
        f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
        f'loss={epoch_means["total_loss"]:.4f}, '
        f'main={epoch_means["main_loss"]:.4f}, '
        f'node={epoch_means["node_loss"]:.4f}, '
        f'lr={history_row["learning_rate"]:.2e}, '
        f'grad={history_row["mean_gradient_norm"]:.3f}, '
        f'time={epoch_seconds / 60.0:.1f} min'
    )
    if validation_result is not None:
        print(
            f'  Validation: mean Dice='
            f'{validation_result["mean_subject_dice"]:.4f}, '
            f'global Dice={validation_result["global_dice"]:.4f}, '
            f'threshold={validation_result["threshold"]:.2f}, '
            f'best={best_mean_dice:.4f} at epoch {best_epoch}'
        )

    atomic_write_json(
        {
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'training_version': TRAINING_VERSION,
            'status': 'running',
            'last_completed_epoch': epoch,
            'global_update_step': global_update_step,
            'best_mean_subject_dice': best_mean_dice,
            'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'validations_without_improvement': (
                validations_without_improvement
            ),
            'total_parameters': EXPECTED_TOTAL_PARAMETERS,
            'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
            'test_targets_read': False,
            'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
            'best_checkpoint_path': (
                str(BEST_CHECKPOINT_PATH)
                if BEST_CHECKPOINT_PATH.exists()
                else ''
            ),
        },
        TRAINING_STATE_PATH,
    )

    if (
        should_validate
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    ):
        stop_reason = 'early_stopping'
        print(
            f'Early stopping after {validations_without_improvement} '
            'validation checks without improvement.'
        )
        break

completed_epoch = history_rows[-1]['epoch'] if history_rows else start_epoch - 1
atomic_write_json(
    {
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'status': 'complete',
        'stop_reason': stop_reason,
        'last_completed_epoch': completed_epoch,
        'global_update_step': global_update_step,
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
        'train_subjects': 102,
        'validation_subjects': 23,
        'test_targets_read': False,
        'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
        'best_checkpoint_path': str(BEST_CHECKPOINT_PATH),
        'history_path': str(HISTORY_PATH),
    },
    TRAINING_STATE_PATH,
)

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET TRAINING COMPLETE')
print('=' * 80)
print(f'Stop reason:                        {stop_reason}')
print(f'Last completed epoch:               {completed_epoch}')
print(f'Best validation mean Dice:          {best_mean_dice:.4f}')
print(f'Best epoch:                         {best_epoch}')
print(f'Frozen validation threshold:        {best_threshold:.2f}')
print(f'Total parameters:                   {parameter_count(training_model):,}')
print(f'Trainable parameters:               {parameter_count(training_model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print('Test targets read:                  False')
print(f'Best checkpoint:                    {BEST_CHECKPOINT_PATH}')
print(f'Last checkpoint:                    {LAST_CHECKPOINT_PATH}')
print(f'Training history:                   {HISTORY_PATH}')
print(f'Training state:                     {TRAINING_STATE_PATH}')


In [ ]:
import sys
import os
import gc
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v2_regularized_epochwise_validation'
CODE_REVISION = 'r2_overfit_control_epochwise_validation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')






print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT')
print('=' * 80)
print('Target access policy: train and validation only; test remains sealed.')

target_rows = []
target_arrays_read_splits = set()
for row in manifest_df[
    manifest_df['dev_split'].isin(['train', 'val'])
].sort_values(['dev_split', 'subject_id']).to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    source_path = Path(str(row['vascufate_v2_npz_path']))
    if not source_path.exists():
        raise FileNotFoundError(f'{subject_id}: frozen NPZ is missing.')
    with np.load(source_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'binary_loss_weight',
            'crop_mask',
            'target_spacing',
            'subject_id',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        spatial_shape = tuple(int(value) for value in target_binary.shape)
        crop_mask = np.asarray(data['crop_mask']) > 0
        binary_loss_weight_value = float(
            np.asarray(data['binary_loss_weight']).item()
        )
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        stored_subject = scalar_string(data['subject_id'])
    target_arrays_read_splits.add(split)
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: stored subject identifier mismatch.')
    if target_soft.shape != target_binary.shape or crop_mask.shape != target_binary.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not set(int(value) for value in np.unique(target_binary)).issubset({0, 1}):
        raise RuntimeError(f'{subject_id}: binary target is not binary.')
    if (
        not np.isfinite(target_soft).all()
        or target_soft.min() < -1e-4
        or target_soft.max() > 1.0001
    ):
        raise RuntimeError(f'{subject_id}: soft target is invalid.')
    if binary_loss_weight_value not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    target_voxels = int(target_binary.sum())
    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
        'spatial_shape': str(spatial_shape),
        'binary_voxels': target_voxels,
        'binary_volume_ml': float(target_voxels * voxel_volume_ml),
        'soft_sum': float(target_soft.sum(dtype=np.float64)),
        'binary_loss_weight': binary_loss_weight_value,
        'target_inside_crop_fraction': 1.0 if target_voxels == 0 else float(
            (target_binary & crop_mask).sum() / target_voxels
        ),
        'all_finite': True,
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_audit_checks = {
    'exactly_102_train_targets_audited': int(
        (target_audit_df['dev_split'] == 'train').sum()
    )
    == 102,
    'exactly_23_validation_targets_audited': int(
        (target_audit_df['dev_split'] == 'val').sum()
    )
    == 23,
    'test_targets_not_opened': target_arrays_read_splits == {'train', 'val'},
    'all_target_arrays_finite': bool(target_audit_df['all_finite'].all()),
    'target_crop_support_fractions_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'train_contains_nonempty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] > 0)
        ).any()
    ),
    'train_contains_empty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] == 0)
        ).any()
    ),
}
if not all(target_audit_checks.values()):
    failed = [
        name for name, passed in target_audit_checks.items() if not passed
    ]
    raise RuntimeError('Target audit failed: ' + ', '.join(failed))

print('\nTarget audit by split:')
display(
    target_audit_df.groupby('dev_split').agg(
        subjects=('subject_id', 'count'),
        empty_targets=('binary_voxels', lambda values: int((values == 0).sum())),
        minimum_volume_ml=('binary_volume_ml', 'min'),
        median_volume_ml=('binary_volume_ml', 'median'),
        maximum_volume_ml=('binary_volume_ml', 'max'),
    ).reset_index()
)
print('\nTarget audit by training lesion bin:')
display(
    target_audit_df[target_audit_df['dev_split'] == 'train']
    .groupby('lesion_volume_bin')
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('binary_volume_ml', 'median'),
    )
    .reset_index()
)






def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)

    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(f'{row["subject_id"]}: atlas labels outside 0..30.')
    return labels, True


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size, spatial_axes):
    source_slices = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source_slices[axis] = spatial_slice
    cropped = array[tuple(source_slices)]
    return center_crop_or_pad(cropped, patch_size, spatial_axes)


def deterministic_patch_center(target, crop_mask, seed, positive_probability=0.75):
    rng = np.random.default_rng(seed)
    positive_coordinates = np.argwhere(target > 0)
    use_positive = (
        len(positive_coordinates) > 0
        and float(rng.random()) < positive_probability
    )
    if use_positive:
        center = positive_coordinates[
            int(rng.integers(0, len(positive_coordinates)))
        ].astype(np.int64)
        jitter = np.asarray([
            int(rng.integers(-16, 17)),
            int(rng.integers(-16, 17)),
            int(rng.integers(-6, 7)),
        ])
        center = center + jitter
        source = 'lesion_jitter'
    else:
        candidates = np.argwhere(crop_mask > 0)
        if len(candidates) == 0:
            center = np.asarray(target.shape) // 2
        else:
            center = candidates[int(rng.integers(0, len(candidates)))]
        source = 'brain_random'
    center = np.clip(center, 0, np.asarray(target.shape) - 1)
    return center, source


manifest_by_subject = {
    str(row['subject_id']): row
    for row in manifest_df.to_dict('records')
}
train_target_df = target_audit_df[target_audit_df['dev_split'] == 'train']
sampling_subjects = []
for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
    matches = train_target_df[
        train_target_df['lesion_volume_bin'] == lesion_bin
    ].sort_values('subject_id')
    if len(matches):
        sampling_subjects.append(str(matches.iloc[0]['subject_id']))
sampling_subjects = list(dict.fromkeys(sampling_subjects))

patch_rows = []
for subject_offset, subject_id in enumerate(sampling_subjects):
    row = manifest_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    for repeat in range(4):
        center, source = deterministic_patch_center(
            target,
            crop_mask,
            SEED + subject_offset * 100 + repeat,
        )
        slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
        patch_target = extract_patch(
            target, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        patch_rows.append({
            'subject_id': subject_id,
            'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
            'repeat': repeat,
            'center_source': source,
            'full_target_voxels': int(target.sum()),
            'patch_target_voxels': int(patch_target.sum()),
            'positive_patch_contains_lesion': (
                source != 'lesion_jitter' or bool(patch_target.any())
            ),
        })

patch_audit_df = pd.DataFrame(patch_rows)
atomic_write_csv(patch_audit_df, PATCH_AUDIT_PATH)
patch_checks = {
    'all_requested_lesion_bins_sampled': len(sampling_subjects) >= 5,
    'four_patches_per_selected_subject': len(patch_audit_df)
    == 4 * len(sampling_subjects),
    'all_lesion_centered_patches_contain_lesion': bool(
        patch_audit_df['positive_patch_contains_lesion'].all()
    ),
    'empty_subject_sampling_supported': bool(
        (patch_audit_df['full_target_voxels'] == 0).any()
    ),
}
if not all(patch_checks.values()):
    failed = [name for name, passed in patch_checks.items() if not passed]
    raise RuntimeError('Patch-sampling checks failed: ' + ', '.join(failed))

print('\nLesion-aware patch audit:')
display(
    patch_audit_df.groupby(
        ['lesion_volume_bin', 'center_source'], dropna=False
    ).agg(
        patches=('subject_id', 'count'),
        minimum_patch_target_voxels=('patch_target_voxels', 'min'),
        maximum_patch_target_voxels=('patch_target_voxels', 'max'),
    ).reset_index()
)






def soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return (1.0 - (2.0 * intersection + epsilon) / (
        denominator + epsilon
    )).mean()


def boundary_weighted_bce(logits, soft_target, binary_target):
    dilated = F.max_pool3d(binary_target, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool3d(
        -binary_target, kernel_size=3, stride=1, padding=1
    )
    boundary = (dilated - eroded).clamp(0.0, 1.0)
    interpolation_uncertainty = 4.0 * soft_target * (1.0 - soft_target)
    weights = 1.0 + 2.0 * boundary + interpolation_uncertainty
    loss = F.binary_cross_entropy_with_logits(
        logits, soft_target, reduction='none'
    )
    return (loss * weights).sum() / weights.sum().clamp_min(1.0)


def focal_tversky_loss(logits, target, alpha=0.50, beta=0.50, gamma=0.75):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return ((1.0 - tversky).clamp_min(0.0) ** gamma)


def segmentation_loss(logits, soft_target, binary_target, hard_loss_weight):
    dice = soft_dice_loss(logits, soft_target)
    boundary_bce = boundary_weighted_bce(logits, soft_target, binary_target)
    hard_tversky_per_case = focal_tversky_loss(logits, binary_target)
    
    
    hard_loss_weight = hard_loss_weight.reshape(-1, 1)
    hard_tversky = (
        hard_tversky_per_case * hard_loss_weight
    ).sum() / hard_loss_weight.sum().clamp_min(1.0)
    total = 0.55 * dice + 0.30 * boundary_bce + 0.15 * hard_tversky
    return total, {
        'soft_dice': dice,
        'boundary_bce': boundary_bce,
        'hard_focal_tversky': hard_tversky,
    }


def territory_presence_target(binary_target, region_map):
    targets = []
    for label in range(1, NUM_TERRITORY_NODES + 1):
        targets.append(
            ((binary_target > 0) & (region_map == label)).any().astype(np.float32)
        )
    return np.asarray(targets, dtype=np.float32)


eligible_real_df = train_target_df[
    (train_target_df['binary_voxels'] > 0)
    & (train_target_df['binary_loss_weight'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_real_df = eligible_real_df[
    eligible_real_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
candidate_midpoint = len(eligible_real_df) // 2
candidate_order = sorted(
    range(len(eligible_real_df)),
    key=lambda index: (abs(index - candidate_midpoint), index),
)
real_subject = None
for candidate_index in candidate_order:
    candidate_subject = str(
        eligible_real_df.iloc[candidate_index]['subject_id']
    )
    candidate_row = manifest_by_subject[candidate_subject]
    with np.load(
        Path(str(candidate_row['vascufate_v2_npz_path'])),
        allow_pickle=False,
    ) as candidate_data:
        candidate_target = np.asarray(
            candidate_data['target_binary'], dtype=np.uint8
        )
    candidate_region, _ = load_region_map(
        candidate_row, candidate_target.shape
    )
    if territory_presence_target(
        candidate_target, candidate_region
    ).sum() > 0:
        real_subject = candidate_subject
        break
if real_subject is None:
    raise RuntimeError(
        'No atlas-enabled training lesion overlaps a valid territory.'
    )
real_row = manifest_by_subject[real_subject]
real_graph = load_graph_inputs(real_row)
if file_sha256(Path(str(real_row['vascufate_v2_npz_path']))) != str(
    real_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{real_subject}: real-loss NPZ hash changed.')
with np.load(
    Path(str(real_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    real_image = np.asarray(data['image'], dtype=np.float32)
    real_binary = np.asarray(data['target_binary'], dtype=np.float32)
    real_soft = np.asarray(data['target_soft'], dtype=np.float32)
    real_crop_mask = np.asarray(data['crop_mask']) > 0
    real_binary_weight = float(np.asarray(data['binary_loss_weight']).item())
real_region, real_atlas_enabled = load_region_map(
    real_row, real_binary.shape
)
real_territory_target = territory_presence_target(real_binary, real_region)
real_center, real_center_source = deterministic_patch_center(
    real_binary, real_crop_mask, SEED + 9001, positive_probability=1.0
)
real_slices = patch_slices_from_center(
    real_binary.shape, real_center, PATCH_SIZE
)
real_image_patch = extract_patch(
    real_image, real_slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
)
real_binary_patch = extract_patch(
    real_binary, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_soft_patch = extract_patch(
    real_soft, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_region_patch = extract_patch(
    real_region, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)

real_batch = {
    'image': torch.from_numpy(real_image_patch[None]).to(device),
    'region_map': torch.from_numpy(real_region_patch[None]).to(device),
    'territory_features': torch.from_numpy(
        real_graph['territory_features'][None]
    ).to(device),
    'territory_feature_valid': torch.from_numpy(
        real_graph['territory_feature_valid'][None]
    ).to(device),
    'territory_valid': torch.from_numpy(
        real_graph['territory_valid'][None]
    ).to(device),
    'territory_reliability': torch.from_numpy(
        real_graph['territory_reliability'][None]
    ).to(device),
    'vessel_features': torch.from_numpy(
        real_graph['vessel_features'][None]
    ).to(device),
    'vessel_feature_valid': torch.from_numpy(
        real_graph['vessel_feature_valid'][None]
    ).to(device),
    'vessel_valid': torch.from_numpy(real_graph['vessel_valid'][None]).to(device),
    'case_features': torch.from_numpy(real_graph['case_features'][None]).to(device),
    'atlas_enabled': torch.as_tensor(
        [real_atlas_enabled], dtype=torch.bool, device=device
    ),
}
real_soft_tensor = torch.from_numpy(real_soft_patch[None, None]).to(device)
real_binary_tensor = torch.from_numpy(real_binary_patch[None, None]).to(device)
real_hard_weight = torch.as_tensor(
    [real_binary_weight], dtype=torch.float32, device=device
)
real_territory_target_tensor = torch.from_numpy(
    real_territory_target[None]
).to(device)


set_determinism(SEED + 1)
model = VascuFateNet(
    schema=graph_schema,
    image_channels=real_image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
amp_scaler = torch.amp.GradScaler(
    device.type, enabled=device.type == 'cuda'
)
before_step = model.segmentation_head.weight.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
    enabled=device.type == 'cuda',
):
    real_output = model(**real_batch, force_atlas_dropout=False)
    main_loss, main_parts = segmentation_loss(
        real_output['logits'],
        real_soft_tensor,
        real_binary_tensor,
        real_hard_weight,
    )
    half_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        real_output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        real_hard_weight,
    )
    quarter_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        real_output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        real_hard_weight,
    )
    node_mask = (
        real_batch['territory_valid'].bool()
        & real_output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        real_output['territory_node_logits'],
        real_territory_target_tensor,
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = real_output['territory_node_logits'].sum() * 0.0
    total_real_loss = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )

amp_scaler.scale(total_real_loss).backward()
amp_scaler.unscale_(optimizer)
real_gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0).item()
)
amp_scaler.step(optimizer)
amp_scaler.update()
optimizer.zero_grad(set_to_none=True)
parameter_update = float(
    (model.segmentation_head.weight.detach() - before_step).abs().max().item()
)

loss_values = {
    'subject_id': real_subject,
    'patch_center_source': real_center_source,
    'patch_target_voxels': int(real_binary_patch.sum()),
    'main_loss': float(main_loss.detach().cpu()),
    'soft_dice_loss': float(main_parts['soft_dice'].detach().cpu()),
    'boundary_bce_loss': float(main_parts['boundary_bce'].detach().cpu()),
    'hard_focal_tversky_loss': float(
        main_parts['hard_focal_tversky'].detach().cpu()
    ),
    'half_resolution_loss': float(half_loss.detach().cpu()),
    'quarter_resolution_loss': float(quarter_loss.detach().cpu()),
    'territory_node_loss': float(node_loss.detach().cpu()),
    'total_loss': float(total_real_loss.detach().cpu()),
    'gradient_norm_before_clipping': real_gradient_norm,
    'segmentation_head_maximum_update': parameter_update,
}
atomic_write_json(loss_values, LOSS_AUDIT_PATH)
loss_checks = {
    'real_patch_contains_lesion': int(real_binary_patch.sum()) > 0,
    'all_real_loss_components_finite': all(
        math.isfinite(float(value))
        for key, value in loss_values.items()
        if key.endswith('_loss') or key == 'total_loss'
    ),
    'real_gradient_norm_positive_and_finite': math.isfinite(real_gradient_norm)
    and real_gradient_norm > 0,
    'optimizer_step_changed_parameters': parameter_update > 0,
    'territory_target_has_30_nodes': real_territory_target.shape == (30,),
    'territory_target_contains_positive_node': real_territory_target.sum() > 0,
    'parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
}
if not all(loss_checks.values()):
    failed = [name for name, passed in loss_checks.items() if not passed]
    raise RuntimeError('Real-loss optimizer preflight failed: ' + ', '.join(failed))

print('\nReal-loss disposable optimizer-step audit:')
display(pd.DataFrame([
    {'metric': name, 'value': value} for name, value in loss_values.items()
]))
print('\nParameter count after model reinitialization:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': parameter_count(model)},
    {'component': 'trainable', 'parameters': parameter_count(model, True)},
]))






def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


@torch.no_grad()
def sliding_window_probability(model, row, image, region_map, graph_inputs):
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, PATCH_SIZE)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(PATCH_SIZE)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, PATCH_SIZE)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    model.eval()
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index : location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(
                np.repeat(graph_inputs['territory_features'][None], count, axis=0)
            ).to(device),
            'territory_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'territory_valid': torch.from_numpy(
                np.repeat(graph_inputs['territory_valid'][None], count, axis=0)
            ).to(device),
            'territory_reliability': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_reliability'][None], count, axis=0
                )
            ).to(device),
            'vessel_features': torch.from_numpy(
                np.repeat(graph_inputs['vessel_features'][None], count, axis=0)
            ).to(device),
            'vessel_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['vessel_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'vessel_valid': torch.from_numpy(
                np.repeat(graph_inputs['vessel_valid'][None], count, axis=0)
            ).to(device),
            'case_features': torch.from_numpy(
                np.repeat(graph_inputs['case_features'][None], count, axis=0)
            ).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **inference_batch, force_atlas_dropout=False
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    return probability[original_slices]


eligible_validation_df = target_audit_df[
    (target_audit_df['dev_split'] == 'val')
    & (target_audit_df['binary_voxels'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_validation_df = eligible_validation_df[
    eligible_validation_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
validation_subject = str(
    eligible_validation_df.iloc[len(eligible_validation_df) // 2]['subject_id']
)
validation_row = manifest_by_subject[validation_subject]
validation_graph = load_graph_inputs(validation_row)
if file_sha256(Path(str(validation_row['vascufate_v2_npz_path']))) != str(
    validation_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{validation_subject}: validation NPZ hash changed.')
with np.load(
    Path(str(validation_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    validation_image = np.asarray(data['image'], dtype=np.float32)
    validation_target = np.asarray(data['target_binary'], dtype=np.uint8)
validation_region, _ = load_region_map(
    validation_row, validation_target.shape
)
validation_probability = sliding_window_probability(
    model,
    validation_row,
    validation_image,
    validation_region,
    validation_graph,
)
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    prediction = validation_probability >= threshold
    intersection = int((prediction & (validation_target > 0)).sum())
    predicted = int(prediction.sum())
    truth = int(validation_target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else (2.0 * intersection) / (predicted + truth)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    precision = 1.0 if predicted == 0 and truth == 0 else (
        intersection / max(predicted, 1)
    )
    threshold_rows.append({
        'subject_id': validation_subject,
        'threshold': threshold,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    })
validation_metric_df = pd.DataFrame(threshold_rows)
atomic_write_csv(validation_metric_df, VALIDATION_AUDIT_PATH)
validation_checks = {
    'validation_probability_shape_matches': validation_probability.shape
    == validation_target.shape,
    'validation_probability_finite': bool(
        np.isfinite(validation_probability).all()
    ),
    'validation_probability_in_unit_interval': bool(
        validation_probability.min() >= 0
        and validation_probability.max() <= 1
    ),
    'five_thresholds_evaluated': len(validation_metric_df) == 5,
    'validation_metrics_finite': bool(
        np.isfinite(
            validation_metric_df[['dice', 'precision', 'recall']].to_numpy()
        ).all()
    ),
    'test_targets_still_not_opened': target_arrays_read_splits == {
        'train',
        'val',
    },
}
if not all(validation_checks.values()):
    failed = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Sliding-window validation preflight failed: ' + ', '.join(failed)
    )

print('\nDisposable full-volume validation smoke metrics:')
display(validation_metric_df)






training_policy = {
    'patch_size': list(PATCH_SIZE),
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'patches_per_subject_per_epoch': 2,
    'sampling_schedule': 'one_lesion_centered_plus_one_brain_context',
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'early_stopping_patience_validations': 10,
    'optimizer': 'AdamW',
    'learning_rate': 1e-4,
    'weight_decay': 5e-4,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'positive_center_probability': 1.0,
    'context_center_probability': 1.0,
    'intensity_gain_range': [0.90, 1.10],
    'intensity_offset_range': [-0.03, 0.03],
    'noise_probability': 0.65,
    'maximum_noise_standard_deviation': 0.02,
    'gamma_probability': 0.50,
    'gamma_range': [0.85, 1.15],
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'main_loss_weights': {
        'soft_dice': 0.55,
        'boundary_weighted_bce': 0.30,
        'hard_focal_tversky': 0.15,
    },
    'focal_tversky_alpha_false_positive': 0.50,
    'focal_tversky_beta_false_negative': 0.50,
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.05,
    'territory_positive_weight': 4.0,
    'validation_inference': '50_percent_overlap_gaussian_sliding_window',
    'threshold_candidates': [
        round(value, 2) for value in np.arange(0.10, 0.61, 0.05)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_training_checks = {
    **frozen_architecture_checks,
    **target_audit_checks,
    **patch_checks,
    **loss_checks,
    **validation_checks,
    'frozen_scaler_hash_matches': scaler_sha256 == EXPECTED_SCALER_SHA256,
    'target_free_model_recheck_passed': preflight_passed,
    'total_parameters_match': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameters_match': parameter_count(model, True)
    == EXPECTED_TOTAL_PARAMETERS,
    'test_split_remains_sealed': 'test' not in target_arrays_read_splits,
}
training_preflight_passed = bool(all(all_training_checks.values()))
atomic_write_csv(
    pd.DataFrame([
        {'check': name, 'passed': passed}
        for name, passed in all_training_checks.items()
    ]),
    PREFLIGHT_AUDIT_PATH,
)
final_gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'training_preflight_passed': training_preflight_passed,
    'full_training_authorized': training_preflight_passed,
    'checks': all_training_checks,
    'training_policy': training_policy,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_sha256': architecture_sha256,
    'total_parameters': parameter_count(model),
    'trainable_parameters': parameter_count(model, True),
    'real_loss_subject': real_subject,
    'validation_smoke_subject': validation_subject,
    'target_arrays_read_splits': sorted(target_arrays_read_splits),
    'test_targets_read': False,
    'study_training_epochs_completed': 0,
}
atomic_write_json(final_gate, PREFLIGHT_GATE_PATH)

print('\nFinal target-aware training preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_training_checks.items()
]))
if not training_preflight_passed:
    failed = [
        name for name, passed in all_training_checks.items() if not passed
    ]
    raise RuntimeError(
        'TRIAL8 TARGET-AWARE TRAINING PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT PASSED')
print('=' * 80)
print('Train targets audited:              102')
print('Validation targets audited:         23')
print('Test targets read:                  False')
print(f'Real-loss subject:                  {real_subject}')
print(f'Validation smoke subject:           {validation_subject}')
print(f'Total parameters:                   {parameter_count(model):,}')
print(f'Trainable parameters:               {parameter_count(model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Frozen scaler SHA-256:              {scaler_sha256}')
print(f'Training preflight gate:            {PREFLIGHT_GATE_PATH}')
print('Study training epochs completed:    0')
print('Full Trial8 training authorized:    True')
print('\nTarget-aware gate passed; starting authorized training below.')








del model, optimizer, amp_scaler, real_output, real_batch
del real_soft_tensor, real_binary_tensor, real_territory_target_tensor
del normal_output, dropped_output, gradient_output, batch
del smoke_loss, total_real_loss, main_loss, half_loss, quarter_loss
del node_loss, node_bce, before_step
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TRAINING_VERSION = 'v2_regularized_balanced_epochwise'
TRAINING_ROOT = (
    PROJECT_ROOT
    / 'checkpoints'
    / 'trial8_vascufate_net_v1'
    / TRAINING_VERSION
)
LAST_CHECKPOINT_PATH = TRAINING_ROOT / 'last_checkpoint.pt'
BEST_CHECKPOINT_PATH = TRAINING_ROOT / 'best_checkpoint.pt'
HISTORY_PATH = TRAINING_ROOT / 'training_history.csv'
VALIDATION_ROOT = TRAINING_ROOT / 'validation'
TRAINING_CONFIG_PATH = TRAINING_ROOT / 'training_config.json'
TRAINING_STATE_PATH = TRAINING_ROOT / 'training_state.json'

assert_project_path_is_safe(TRAINING_ROOT)
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

final_training_gate = json.loads(
    PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if final_training_gate.get('training_preflight_passed') is not True:
    raise RuntimeError('Target-aware training preflight is not passed.')
if final_training_gate.get('full_training_authorized') is not True:
    raise RuntimeError('Full Trial8 training is not authorized.')
if final_training_gate.get('test_targets_read') is not False:
    raise RuntimeError('Training preflight reports test-target access.')
if final_training_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Training gate parameter count changed.')
if final_training_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Training gate scaler hash changed.')

policy = final_training_gate['training_policy']
BATCH_SIZE = int(policy['batch_size'])
ACCUMULATION_STEPS = int(policy['gradient_accumulation_steps'])
MAX_EPOCHS = int(policy['maximum_epochs'])
VALIDATION_INTERVAL = int(policy['validation_interval_epochs'])
PATIENCE_VALIDATIONS = int(policy['early_stopping_patience_validations'])
BASE_LEARNING_RATE = float(policy['learning_rate'])
WEIGHT_DECAY = float(policy['weight_decay'])
GRADIENT_CLIP_NORM = float(policy['gradient_clip_norm'])
POSITIVE_CENTER_PROBABILITY = float(policy['positive_center_probability'])
PATCHES_PER_SUBJECT = int(policy['patches_per_subject_per_epoch'])
GRAPH_FEATURE_MASK_PROBABILITY = float(
    policy['graph_feature_mask_probability']
)
THRESHOLDS = [float(value) for value in policy['threshold_candidates']]
NUM_WORKERS = 2
IMPROVEMENT_EPSILON = 1e-4

if tuple(policy['patch_size']) != PATCH_SIZE:
    raise RuntimeError('Frozen training patch size changed.')
if int(policy['effective_batch_size']) != BATCH_SIZE * ACCUMULATION_STEPS:
    raise RuntimeError('Frozen effective batch size is inconsistent.')
if PATCHES_PER_SUBJECT != 2:
    raise RuntimeError('Regularized sampling requires two patches per subject.')
if VALIDATION_INTERVAL != 1:
    raise RuntimeError('Regularized training must validate every epoch.')


def atomic_torch_save(record, path):
    path = Path(path)
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


def augment_training_image(image, rng):
    augmented = image.astype(np.float32, copy=True)
    intensity_channels = min(13, augmented.shape[0])
    gains = rng.uniform(
        policy['intensity_gain_range'][0],
        policy['intensity_gain_range'][1],
        size=(intensity_channels, 1, 1, 1),
    )
    offsets = rng.uniform(
        policy['intensity_offset_range'][0],
        policy['intensity_offset_range'][1],
        size=(intensity_channels, 1, 1, 1),
    )
    augmented[:intensity_channels] = (
        augmented[:intensity_channels] * gains + offsets
    )
    if float(rng.random()) < float(policy['noise_probability']):
        noise_scale = float(rng.uniform(
            0.0, policy['maximum_noise_standard_deviation']
        ))
        noise = rng.normal(
            0.0,
            noise_scale,
            size=augmented[:intensity_channels].shape,
        ).astype(np.float32)
        augmented[:intensity_channels] += noise
    non_delta_end = min(9, intensity_channels)
    augmented[:non_delta_end] = np.clip(
        augmented[:non_delta_end], 0.0, 1.0
    )
    if (
        non_delta_end > 0
        and float(rng.random()) < float(policy['gamma_probability'])
    ):
        gamma = float(rng.uniform(*policy['gamma_range']))
        augmented[:non_delta_end] = np.power(
            augmented[:non_delta_end], gamma
        ).astype(np.float32)
    if intensity_channels > 9:
        augmented[9:intensity_channels] = np.clip(
            augmented[9:intensity_channels], -1.0, 1.0
        )
    return augmented


class Trial8PatchDataset(torch.utils.data.Dataset):
    def __init__(self, rows, base_seed):
        self.rows = list(rows)
        self.base_seed = int(base_seed)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.rows) * PATCHES_PER_SUBJECT

    def __getitem__(self, index):
        row_index = index // PATCHES_PER_SUBJECT
        sample_slot = index % PATCHES_PER_SUBJECT
        row = self.rows[row_index]
        subject_id = str(row['subject_id'])
        sample_seed = (
            self.base_seed
            + self.epoch * 100_003
            + row_index * 997
            + sample_slot * 53
        )
        rng = np.random.default_rng(
            sample_seed
        )
        with np.load(
            Path(str(row['vascufate_v2_npz_path'])),
            allow_pickle=False,
        ) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target_binary = np.asarray(
                data['target_binary'], dtype=np.float32
            )
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            crop_mask = np.asarray(data['crop_mask']) > 0
            binary_loss_weight = float(
                np.asarray(data['binary_loss_weight']).item()
            )
        region_map, atlas_enabled = load_region_map(
            row, target_binary.shape
        )
        graph = load_graph_inputs(row)
        graph = {
            key: np.asarray(value).copy()
            if isinstance(value, np.ndarray)
            else value
            for key, value in graph.items()
        }
        territory_feature_mask = (
            rng.random(graph['territory_features'].shape)
            < GRAPH_FEATURE_MASK_PROBABILITY
        ) & (graph['territory_feature_valid'] > 0)
        vessel_feature_mask = (
            rng.random(graph['vessel_features'].shape)
            < GRAPH_FEATURE_MASK_PROBABILITY
        ) & (graph['vessel_feature_valid'] > 0)
        graph['territory_features'][territory_feature_mask] = 0.0
        graph['territory_feature_valid'][territory_feature_mask] = 0.0
        graph['vessel_features'][vessel_feature_mask] = 0.0
        graph['vessel_feature_valid'][vessel_feature_mask] = 0.0
        territory_target = territory_presence_target(
            target_binary, region_map
        )
        center, center_source = deterministic_patch_center(
            target_binary,
            crop_mask,
            sample_seed,
            positive_probability=(
                POSITIVE_CENTER_PROBABILITY if sample_slot == 0 else 0.0
            ),
        )
        slices = patch_slices_from_center(
            target_binary.shape, center, PATCH_SIZE
        )
        image_patch = extract_patch(
            image, slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
        )
        binary_patch = extract_patch(
            target_binary, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        soft_patch = extract_patch(
            target_soft, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        region_patch = extract_patch(
            region_map, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        image_patch = augment_training_image(image_patch, rng)
        return {
            'subject_id': subject_id,
            'center_source': center_source,
            'sample_slot': sample_slot,
            'image': torch.from_numpy(image_patch),
            'region_map': torch.from_numpy(region_patch),
            'target_binary': torch.from_numpy(binary_patch[None]),
            'target_soft': torch.from_numpy(soft_patch[None]),
            'binary_loss_weight': torch.as_tensor(
                binary_loss_weight, dtype=torch.float32
            ),
            'territory_target': torch.from_numpy(territory_target),
            'territory_features': torch.from_numpy(
                graph['territory_features']
            ),
            'territory_feature_valid': torch.from_numpy(
                graph['territory_feature_valid']
            ),
            'territory_valid': torch.from_numpy(graph['territory_valid']),
            'territory_reliability': torch.from_numpy(
                graph['territory_reliability']
            ),
            'vessel_features': torch.from_numpy(graph['vessel_features']),
            'vessel_feature_valid': torch.from_numpy(
                graph['vessel_feature_valid']
            ),
            'vessel_valid': torch.from_numpy(graph['vessel_valid']),
            'case_features': torch.from_numpy(graph['case_features']),
            'atlas_enabled': torch.as_tensor(
                atlas_enabled, dtype=torch.bool
            ),
        }


def to_device(batch, device):
    return {
        key: value.to(device, non_blocking=True)
        if torch.is_tensor(value)
        else value
        for key, value in batch.items()
    }


def compute_training_loss(model, batch):
    model_inputs = {
        key: batch[key]
        for key in [
            'image',
            'region_map',
            'territory_features',
            'territory_feature_valid',
            'territory_valid',
            'territory_reliability',
            'vessel_features',
            'vessel_feature_valid',
            'vessel_valid',
            'case_features',
            'atlas_enabled',
        ]
    }
    output = model(**model_inputs)
    main_loss, main_parts = segmentation_loss(
        output['logits'],
        batch['target_soft'],
        batch['target_binary'],
        batch['binary_loss_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        batch['binary_loss_weight'],
    )
    quarter_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        batch['binary_loss_weight'],
    )
    node_mask = (
        batch['territory_valid'].bool()
        & output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        output['territory_node_logits'],
        batch['territory_target'],
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=batch['image'].device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = output['territory_node_logits'].sum() * 0.0
    total = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )
    parts = {
        'total_loss': total,
        'main_loss': main_loss,
        'soft_dice_loss': main_parts['soft_dice'],
        'boundary_bce_loss': main_parts['boundary_bce'],
        'hard_focal_tversky_loss': main_parts['hard_focal_tversky'],
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
    }
    return total, parts


def epoch_loader(dataset, epoch):
    dataset.set_epoch(epoch)
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda',
        persistent_workers=False,
        drop_last=False,
        generator=generator,
    )


def subject_metrics(probability, target, threshold):
    prediction = probability >= threshold
    target = target > 0
    intersection = int((prediction & target).sum())
    predicted = int(prediction.sum())
    truth = int(target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else 2.0 * intersection / (predicted + truth)
    )
    precision = (
        1.0
        if predicted == 0 and truth == 0
        else intersection / max(predicted, 1)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    return {
        'dice': float(dice),
        'precision': float(precision),
        'recall': float(recall),
        'intersection': intersection,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    }


@torch.no_grad()
def validate_all_subjects(model, validation_rows, epoch):
    threshold_rows = []
    model.eval()
    for case_index, row in enumerate(validation_rows, start=1):
        subject_id = str(row['subject_id'])
        with np.load(
            Path(str(row['vascufate_v2_npz_path'])),
            allow_pickle=False,
        ) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target = np.asarray(data['target_binary'], dtype=np.uint8)
            spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        region_map, _ = load_region_map(row, target.shape)
        graph_inputs = load_graph_inputs(row)
        probability = sliding_window_probability(
            model, row, image, region_map, graph_inputs
        )
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        for threshold in THRESHOLDS:
            metrics = subject_metrics(probability, target, threshold)
            threshold_rows.append({
                'epoch': epoch,
                'subject_id': subject_id,
                'lesion_volume_bin': str(
                    row.get('lesion_volume_bin', 'unknown')
                ),
                'threshold': threshold,
                **metrics,
                'predicted_volume_ml': (
                    metrics['predicted_voxels'] * voxel_volume_ml
                ),
                'target_volume_ml': metrics['target_voxels'] * voxel_volume_ml,
                'absolute_volume_error_ml': abs(
                    metrics['predicted_voxels'] - metrics['target_voxels']
                )
                * voxel_volume_ml,
            })
        if case_index == 1 or case_index % 5 == 0 or case_index == len(validation_rows):
            print(
                f'  Validated {case_index}/{len(validation_rows)}: '
                f'{subject_id}'
            )
    case_frame = pd.DataFrame(threshold_rows)
    summary = (
        case_frame.groupby('threshold')
        .agg(
            mean_subject_dice=('dice', 'mean'),
            median_subject_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_error_ml=('absolute_volume_error_ml', 'mean'),
        )
        .reset_index()
    )
    global_rows = []
    for threshold, group in case_frame.groupby('threshold'):
        intersection = int(group['intersection'].sum())
        predicted = int(group['predicted_voxels'].sum())
        truth = int(group['target_voxels'].sum())
        global_rows.append({
            'threshold': float(threshold),
            'global_dice': (
                1.0
                if predicted + truth == 0
                else 2.0 * intersection / (predicted + truth)
            ),
        })
    summary = summary.merge(pd.DataFrame(global_rows), on='threshold')
    summary = summary.sort_values(
        ['mean_subject_dice', 'mean_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = summary.iloc[0].to_dict()
    selected_threshold = float(selected['threshold'])
    selected_cases = case_frame[
        np.isclose(case_frame['threshold'], selected_threshold)
    ].copy()
    lesion_bin_summary = (
        selected_cases.groupby('lesion_volume_bin', dropna=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_dice=('dice', 'mean'),
            median_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_error_ml=(
                'absolute_volume_error_ml', 'mean'
            ),
        )
        .reset_index()
        .sort_values('lesion_volume_bin')
    )
    epoch_directory = VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_directory.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(case_frame, epoch_directory / 'validation_cases.csv')
    atomic_write_csv(summary, epoch_directory / 'threshold_summary.csv')
    atomic_write_csv(
        lesion_bin_summary,
        epoch_directory / 'lesion_bin_summary.csv',
    )
    atomic_write_json(
        {
            'epoch': epoch,
            'selected_threshold': float(selected['threshold']),
            'mean_subject_dice': float(selected['mean_subject_dice']),
            'median_subject_dice': float(selected['median_subject_dice']),
            'global_dice': float(selected['global_dice']),
            'mean_precision': float(selected['mean_precision']),
            'mean_recall': float(selected['mean_recall']),
            'mean_absolute_volume_error_ml': float(
                selected['mean_absolute_volume_error_ml']
            ),
            'lesion_bin_mean_dice': {
                str(row['lesion_volume_bin']): float(row['mean_dice'])
                for row in lesion_bin_summary.to_dict('records')
            },
            'test_targets_read': False,
        },
        epoch_directory / 'validation_summary.json',
    )
    selected['lesion_bin_mean_dice'] = {
        str(row['lesion_volume_bin']): float(row['mean_dice'])
        for row in lesion_bin_summary.to_dict('records')
    }
    return selected


train_rows = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .to_dict('records')
)
validation_rows = (
    manifest_df[manifest_df['dev_split'] == 'val']
    .sort_values('subject_id')
    .to_dict('records')
)
if len(train_rows) != 102 or len(validation_rows) != 23:
    raise RuntimeError('Training/validation split count changed.')

training_dataset = Trial8PatchDataset(train_rows, SEED + 10_000)
regularization_checks = {
    'validation_runs_every_epoch': VALIDATION_INTERVAL == 1,
    'two_patches_per_subject_per_epoch': PATCHES_PER_SUBJECT == 2,
    'dataset_has_204_patches_per_epoch': len(training_dataset) == 204,
    'weight_decay_increased': WEIGHT_DECAY == 5e-4,
    'learning_rate_reduced': BASE_LEARNING_RATE == 1e-4,
    'balanced_tversky_penalties': (
        float(policy['focal_tversky_alpha_false_positive']) == 0.50
        and float(policy['focal_tversky_beta_false_negative']) == 0.50
    ),
    'graph_feature_masking_enabled': (
        GRAPH_FEATURE_MASK_PROBABILITY == 0.05
    ),
    'test_access_remains_disabled': (
        policy['test_access_during_training'] is False
    ),
}
if not all(regularization_checks.values()):
    failed = [
        name for name, passed in regularization_checks.items() if not passed
    ]
    raise RuntimeError(
        'Regularized training policy check failed: ' + ', '.join(failed)
    )
print('\nRegularized training-policy checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in regularization_checks.items()
]))

updates_per_epoch = math.ceil(
    math.ceil(len(training_dataset) / BATCH_SIZE) / ACCUMULATION_STEPS
)
total_update_steps = MAX_EPOCHS * updates_per_epoch
warmup_steps = 3 * updates_per_epoch


def learning_rate_multiplier(step):
    if step < warmup_steps:
        return float(step + 1) / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(
        total_update_steps - warmup_steps, 1
    )
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


set_determinism(SEED + 2)
training_model = VascuFateNet(
    schema=graph_schema,
    image_channels=15,
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
if parameter_count(training_model) != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Fresh training model parameter count changed.')

training_optimizer = torch.optim.AdamW(
    training_model.parameters(),
    lr=BASE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
training_scheduler = torch.optim.lr_scheduler.LambdaLR(
    training_optimizer,
    lr_lambda=learning_rate_multiplier,
)
training_amp_scaler = torch.amp.GradScaler(
    device.type, enabled=device.type == 'cuda'
)

start_epoch = 1
global_update_step = 0
best_mean_dice = -1.0
best_epoch = 0
best_threshold = 0.20
validations_without_improvement = 0
history_rows = []

if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    if checkpoint.get('training_version') != TRAINING_VERSION:
        raise RuntimeError('Existing checkpoint training version mismatch.')
    if checkpoint.get('verified_manifest_sha256') != verified_manifest_sha256:
        raise RuntimeError('Existing checkpoint manifest hash mismatch.')
    if checkpoint.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
        raise RuntimeError('Existing checkpoint scaler hash mismatch.')
    if checkpoint.get('graph_schema_sha256') != safe_schema_sha256:
        raise RuntimeError('Existing checkpoint graph-schema hash mismatch.')
    if checkpoint.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
        raise RuntimeError('Existing checkpoint parameter count mismatch.')
    training_model.load_state_dict(checkpoint['model_state_dict'])
    training_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    training_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    training_amp_scaler.load_state_dict(checkpoint['amp_scaler_state_dict'])
    restore_rng_state(checkpoint['rng_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    global_update_step = int(checkpoint['global_update_step'])
    best_mean_dice = float(checkpoint['best_mean_dice'])
    best_epoch = int(checkpoint['best_epoch'])
    best_threshold = float(checkpoint['best_threshold'])
    validations_without_improvement = int(
        checkpoint['validations_without_improvement']
    )
    history_rows = list(checkpoint.get('history_rows', []))
    print(
        f'Resuming Trial8 after epoch {start_epoch - 1}; '
        f'best Dice={best_mean_dice:.4f} at epoch {best_epoch}.'
    )
else:
    print('Starting Trial8 from a fresh deterministic initialization.')

training_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_version': TRAINING_VERSION,
    'training_policy': policy,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_sha256': EXPECTED_SCALER_SHA256,
    'graph_schema_sha256': safe_schema_sha256,
    'architecture_sha256': architecture_sha256,
    'total_parameters': parameter_count(training_model),
    'trainable_parameters': parameter_count(training_model, True),
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
    },
    'train_subjects': 102,
    'validation_subjects': 23,
    'test_subjects_accessed': 0,
}
if TRAINING_CONFIG_PATH.exists():
    existing_config = json.loads(
        TRAINING_CONFIG_PATH.read_text(encoding='utf-8')
    )
    immutable_keys = [
        'training_version',
        'verified_manifest_sha256',
        'scaler_sha256',
        'graph_schema_sha256',
        'architecture_sha256',
        'total_parameters',
        'training_policy',
    ]
    if any(
        existing_config.get(key) != training_config.get(key)
        for key in immutable_keys
    ):
        raise RuntimeError('Existing training configuration changed.')
else:
    atomic_write_json(training_config, TRAINING_CONFIG_PATH)

print('\n' + '=' * 80)
print('TRIAL8 RESUME-SAFE TRAINING START')
print('=' * 80)
print(f'Device:                             {device}')
if device.type == 'cuda':
    print(f'GPU:                                {torch.cuda.get_device_name(0)}')
print(f'Train subjects:                     {len(train_rows)}')
print(f'Validation subjects:                {len(validation_rows)}')
print('Test subjects accessed:             0')
print(f'Patch size:                         {PATCH_SIZE}')
print(f'Patches per subject per epoch:      {PATCHES_PER_SUBJECT}')
print(f'Patches per epoch:                  {len(training_dataset)}')
print(f'Batch / accumulation / effective:   {BATCH_SIZE} / {ACCUMULATION_STEPS} / {BATCH_SIZE * ACCUMULATION_STEPS}')
print(f'Updates per epoch:                  {updates_per_epoch}')
print(f'Maximum epochs:                     {MAX_EPOCHS}')
print(f'Validation interval:                every {VALIDATION_INTERVAL} epoch')
print(f'Early-stopping patience:            {PATIENCE_VALIDATIONS} epochs')
print(f'Learning rate / weight decay:       {BASE_LEARNING_RATE:.1e} / {WEIGHT_DECAY:.1e}')
print(f'Total parameters:                   {parameter_count(training_model):,}')
print(f'Trainable parameters:               {parameter_count(training_model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Checkpoint directory:               {TRAINING_ROOT}')

training_already_complete = (
    start_epoch > MAX_EPOCHS
    or validations_without_improvement >= PATIENCE_VALIDATIONS
)
stop_reason = (
    'already_complete_on_resume'
    if training_already_complete
    else 'maximum_epochs_reached'
)
if training_already_complete:
    print(
        'The latest checkpoint already satisfies a terminal stopping '
        'condition; no additional epochs will run.'
    )

for epoch in range(
    start_epoch,
    start_epoch if training_already_complete else MAX_EPOCHS + 1,
):
    epoch_start = datetime.now(timezone.utc)
    loader = epoch_loader(training_dataset, epoch)
    training_model.train()
    training_optimizer.zero_grad(set_to_none=True)
    running = {
        'total_loss': 0.0,
        'main_loss': 0.0,
        'soft_dice_loss': 0.0,
        'boundary_bce_loss': 0.0,
        'hard_focal_tversky_loss': 0.0,
        'half_loss': 0.0,
        'quarter_loss': 0.0,
        'node_loss': 0.0,
        'atlas_retained_fraction': 0.0,
    }
    processed_batches = 0
    processed_samples = 0
    positive_patch_samples = 0
    lesion_centered_samples = 0
    epoch_gradient_norms = []
    for batch_index, cpu_batch in enumerate(loader, start=1):
        batch_samples = int(cpu_batch['target_binary'].shape[0])
        processed_samples += batch_samples
        positive_patch_samples += int(
            (
                cpu_batch['target_binary'].reshape(batch_samples, -1).sum(1)
                > 0
            ).sum().item()
        )
        lesion_centered_samples += sum(
            str(value) == 'lesion_jitter'
            for value in cpu_batch['center_source']
        )
        batch = to_device(cpu_batch, device)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            total_loss, parts = compute_training_loss(training_model, batch)
            accumulation_group_start = (
                (batch_index - 1) // ACCUMULATION_STEPS
            ) * ACCUMULATION_STEPS
            accumulation_group_size = min(
                ACCUMULATION_STEPS,
                len(loader) - accumulation_group_start,
            )
            scaled_loss = total_loss / accumulation_group_size
        training_amp_scaler.scale(scaled_loss).backward()
        for name in running:
            running[name] += float(parts[name].detach().cpu())
        processed_batches += 1
        should_update = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(loader)
        )
        if should_update:
            training_amp_scaler.unscale_(training_optimizer)
            gradient_norm = float(
                torch.nn.utils.clip_grad_norm_(
                    training_model.parameters(), GRADIENT_CLIP_NORM
                ).item()
            )
            if not math.isfinite(gradient_norm):
                raise RuntimeError(
                    f'Non-finite gradient norm at epoch {epoch}.'
                )
            epoch_gradient_norms.append(gradient_norm)
            training_amp_scaler.step(training_optimizer)
            training_amp_scaler.update()
            training_optimizer.zero_grad(set_to_none=True)
            training_scheduler.step()
            global_update_step += 1
    epoch_means = {
        name: value / max(processed_batches, 1)
        for name, value in running.items()
    }
    if not all(math.isfinite(value) for value in epoch_means.values()):
        raise RuntimeError(f'Non-finite training loss at epoch {epoch}.')

    validation_result = None
    improved = False
    should_validate = (
        epoch % VALIDATION_INTERVAL == 0 or epoch == MAX_EPOCHS
    )
    if should_validate:
        print(f'Running full validation at epoch {epoch}...')
        validation_result = validate_all_subjects(
            training_model, validation_rows, epoch
        )
        current_dice = float(validation_result['mean_subject_dice'])
        current_threshold = float(validation_result['threshold'])
        if current_dice > best_mean_dice + IMPROVEMENT_EPSILON:
            improved = True
            best_mean_dice = current_dice
            best_epoch = epoch
            best_threshold = current_threshold
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1

    epoch_seconds = (
        datetime.now(timezone.utc) - epoch_start
    ).total_seconds()
    history_row = {
        'epoch': epoch,
        'global_update_step': global_update_step,
        'learning_rate': float(training_optimizer.param_groups[0]['lr']),
        **epoch_means,
        'mean_gradient_norm': float(np.mean(epoch_gradient_norms)),
        'maximum_gradient_norm': float(np.max(epoch_gradient_norms)),
        'positive_patch_fraction': (
            positive_patch_samples / max(processed_samples, 1)
        ),
        'lesion_centered_fraction': (
            lesion_centered_samples / max(processed_samples, 1)
        ),
        'validation_performed': should_validate,
        'validation_threshold': (
            float(validation_result['threshold'])
            if validation_result is not None
            else np.nan
        ),
        'validation_mean_subject_dice': (
            float(validation_result['mean_subject_dice'])
            if validation_result is not None
            else np.nan
        ),
        'validation_global_dice': (
            float(validation_result['global_dice'])
            if validation_result is not None
            else np.nan
        ),
        'validation_median_subject_dice': (
            float(validation_result['median_subject_dice'])
            if validation_result is not None
            else np.nan
        ),
        'validation_mean_precision': (
            float(validation_result['mean_precision'])
            if validation_result is not None
            else np.nan
        ),
        'validation_mean_recall': (
            float(validation_result['mean_recall'])
            if validation_result is not None
            else np.nan
        ),
        'validation_mean_absolute_volume_error_ml': (
            float(validation_result['mean_absolute_volume_error_ml'])
            if validation_result is not None
            else np.nan
        ),
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'improved': improved,
        'epoch_seconds': epoch_seconds,
        'test_targets_read': False,
    }
    if validation_result is not None:
        for lesion_bin, value in validation_result[
            'lesion_bin_mean_dice'
        ].items():
            history_row[
                f'validation_mean_dice_{lesion_bin}'
            ] = float(value)
    history_rows.append(to_builtin(history_row))
    atomic_write_csv(pd.DataFrame(history_rows), HISTORY_PATH)

    checkpoint_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'epoch': epoch,
        'global_update_step': global_update_step,
        'model_state_dict': training_model.state_dict(),
        'optimizer_state_dict': training_optimizer.state_dict(),
        'scheduler_state_dict': training_scheduler.state_dict(),
        'amp_scaler_state_dict': training_amp_scaler.state_dict(),
        'rng_state': capture_rng_state(),
        'best_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'history_rows': history_rows,
        'verified_manifest_sha256': verified_manifest_sha256,
        'scaler_sha256': EXPECTED_SCALER_SHA256,
        'graph_schema_sha256': safe_schema_sha256,
        'architecture_sha256': architecture_sha256,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'test_targets_read': False,
    }
    atomic_torch_save(checkpoint_record, LAST_CHECKPOINT_PATH)
    if improved:
        atomic_torch_save(checkpoint_record, BEST_CHECKPOINT_PATH)

    print(
        f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
        f'loss={epoch_means["total_loss"]:.4f}, '
        f'main={epoch_means["main_loss"]:.4f}, '
        f'node={epoch_means["node_loss"]:.4f}, '
        f'lr={history_row["learning_rate"]:.2e}, '
        f'grad={history_row["mean_gradient_norm"]:.3f}, '
        f'positive patches={history_row["positive_patch_fraction"]:.3f}, '
        f'time={epoch_seconds / 60.0:.1f} min'
    )
    if validation_result is not None:
        print(
            f'  Validation: mean Dice='
            f'{validation_result["mean_subject_dice"]:.4f}, '
            f'median Dice={validation_result["median_subject_dice"]:.4f}, '
            f'global Dice={validation_result["global_dice"]:.4f}, '
            f'precision={validation_result["mean_precision"]:.4f}, '
            f'recall={validation_result["mean_recall"]:.4f}, '
            f'threshold={validation_result["threshold"]:.2f}, '
            f'best={best_mean_dice:.4f} at epoch {best_epoch}'
        )
        bin_text = ', '.join(
            f'{name}={value:.4f}'
            for name, value in validation_result[
                'lesion_bin_mean_dice'
            ].items()
        )
        print(f'  Mean Dice by lesion bin: {bin_text}')

    atomic_write_json(
        {
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'training_version': TRAINING_VERSION,
            'status': 'running',
            'last_completed_epoch': epoch,
            'global_update_step': global_update_step,
            'best_mean_subject_dice': best_mean_dice,
            'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'validations_without_improvement': (
                validations_without_improvement
            ),
            'total_parameters': EXPECTED_TOTAL_PARAMETERS,
            'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
            'test_targets_read': False,
            'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
            'best_checkpoint_path': (
                str(BEST_CHECKPOINT_PATH)
                if BEST_CHECKPOINT_PATH.exists()
                else ''
            ),
        },
        TRAINING_STATE_PATH,
    )

    if (
        should_validate
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    ):
        stop_reason = 'early_stopping'
        print(
            f'Early stopping after {validations_without_improvement} '
            'validation checks without improvement.'
        )
        break

completed_epoch = history_rows[-1]['epoch'] if history_rows else start_epoch - 1
atomic_write_json(
    {
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'status': 'complete',
        'stop_reason': stop_reason,
        'last_completed_epoch': completed_epoch,
        'global_update_step': global_update_step,
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
        'train_subjects': 102,
        'validation_subjects': 23,
        'test_targets_read': False,
        'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
        'best_checkpoint_path': str(BEST_CHECKPOINT_PATH),
        'history_path': str(HISTORY_PATH),
    },
    TRAINING_STATE_PATH,
)

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET TRAINING COMPLETE')
print('=' * 80)
print(f'Stop reason:                        {stop_reason}')
print(f'Last completed epoch:               {completed_epoch}')
print(f'Best validation mean Dice:          {best_mean_dice:.4f}')
print(f'Best epoch:                         {best_epoch}')
print(f'Frozen validation threshold:        {best_threshold:.2f}')
print(f'Total parameters:                   {parameter_count(training_model):,}')
print(f'Trainable parameters:               {parameter_count(training_model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print('Test targets read:                  False')
print('Validation cadence:                 every epoch')
print(f'Patches per subject per epoch:      {PATCHES_PER_SUBJECT}')
print(f'Learning rate / weight decay:       {BASE_LEARNING_RATE:.1e} / {WEIGHT_DECAY:.1e}')
print(f'Best checkpoint:                    {BEST_CHECKPOINT_PATH}')
print(f'Last checkpoint:                    {LAST_CHECKPOINT_PATH}')
print(f'Training history:                   {HISTORY_PATH}')
print(f'Training state:                     {TRAINING_STATE_PATH}')


In [ ]:
import sys
import os
import gc
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v2_regularized_epochwise_validation'
CODE_REVISION = 'r2_overfit_control_epochwise_validation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')






print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT')
print('=' * 80)
print('Target access policy: train and validation only; test remains sealed.')

target_rows = []
target_arrays_read_splits = set()
for row in manifest_df[
    manifest_df['dev_split'].isin(['train', 'val'])
].sort_values(['dev_split', 'subject_id']).to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    source_path = Path(str(row['vascufate_v2_npz_path']))
    if not source_path.exists():
        raise FileNotFoundError(f'{subject_id}: frozen NPZ is missing.')
    with np.load(source_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'binary_loss_weight',
            'crop_mask',
            'target_spacing',
            'subject_id',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        spatial_shape = tuple(int(value) for value in target_binary.shape)
        crop_mask = np.asarray(data['crop_mask']) > 0
        binary_loss_weight_value = float(
            np.asarray(data['binary_loss_weight']).item()
        )
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        stored_subject = scalar_string(data['subject_id'])
    target_arrays_read_splits.add(split)
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: stored subject identifier mismatch.')
    if target_soft.shape != target_binary.shape or crop_mask.shape != target_binary.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not set(int(value) for value in np.unique(target_binary)).issubset({0, 1}):
        raise RuntimeError(f'{subject_id}: binary target is not binary.')
    if (
        not np.isfinite(target_soft).all()
        or target_soft.min() < -1e-4
        or target_soft.max() > 1.0001
    ):
        raise RuntimeError(f'{subject_id}: soft target is invalid.')
    if binary_loss_weight_value not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    target_voxels = int(target_binary.sum())
    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
        'spatial_shape': str(spatial_shape),
        'binary_voxels': target_voxels,
        'binary_volume_ml': float(target_voxels * voxel_volume_ml),
        'soft_sum': float(target_soft.sum(dtype=np.float64)),
        'binary_loss_weight': binary_loss_weight_value,
        'target_inside_crop_fraction': 1.0 if target_voxels == 0 else float(
            (target_binary & crop_mask).sum() / target_voxels
        ),
        'all_finite': True,
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_audit_checks = {
    'exactly_102_train_targets_audited': int(
        (target_audit_df['dev_split'] == 'train').sum()
    )
    == 102,
    'exactly_23_validation_targets_audited': int(
        (target_audit_df['dev_split'] == 'val').sum()
    )
    == 23,
    'test_targets_not_opened': target_arrays_read_splits == {'train', 'val'},
    'all_target_arrays_finite': bool(target_audit_df['all_finite'].all()),
    'target_crop_support_fractions_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'train_contains_nonempty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] > 0)
        ).any()
    ),
    'train_contains_empty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] == 0)
        ).any()
    ),
}
if not all(target_audit_checks.values()):
    failed = [
        name for name, passed in target_audit_checks.items() if not passed
    ]
    raise RuntimeError('Target audit failed: ' + ', '.join(failed))

print('\nTarget audit by split:')
display(
    target_audit_df.groupby('dev_split').agg(
        subjects=('subject_id', 'count'),
        empty_targets=('binary_voxels', lambda values: int((values == 0).sum())),
        minimum_volume_ml=('binary_volume_ml', 'min'),
        median_volume_ml=('binary_volume_ml', 'median'),
        maximum_volume_ml=('binary_volume_ml', 'max'),
    ).reset_index()
)
print('\nTarget audit by training lesion bin:')
display(
    target_audit_df[target_audit_df['dev_split'] == 'train']
    .groupby('lesion_volume_bin')
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('binary_volume_ml', 'median'),
    )
    .reset_index()
)






def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)

    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(f'{row["subject_id"]}: atlas labels outside 0..30.')
    return labels, True


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size, spatial_axes):
    source_slices = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source_slices[axis] = spatial_slice
    cropped = array[tuple(source_slices)]
    return center_crop_or_pad(cropped, patch_size, spatial_axes)


def deterministic_patch_center(target, crop_mask, seed, positive_probability=0.75):
    rng = np.random.default_rng(seed)
    positive_coordinates = np.argwhere(target > 0)
    use_positive = (
        len(positive_coordinates) > 0
        and float(rng.random()) < positive_probability
    )
    if use_positive:
        center = positive_coordinates[
            int(rng.integers(0, len(positive_coordinates)))
        ].astype(np.int64)
        jitter = np.asarray([
            int(rng.integers(-16, 17)),
            int(rng.integers(-16, 17)),
            int(rng.integers(-6, 7)),
        ])
        center = center + jitter
        source = 'lesion_jitter'
    else:
        candidates = np.argwhere(crop_mask > 0)
        if len(candidates) == 0:
            center = np.asarray(target.shape) // 2
        else:
            center = candidates[int(rng.integers(0, len(candidates)))]
        source = 'brain_random'
    center = np.clip(center, 0, np.asarray(target.shape) - 1)
    return center, source


manifest_by_subject = {
    str(row['subject_id']): row
    for row in manifest_df.to_dict('records')
}
train_target_df = target_audit_df[target_audit_df['dev_split'] == 'train']
sampling_subjects = []
for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
    matches = train_target_df[
        train_target_df['lesion_volume_bin'] == lesion_bin
    ].sort_values('subject_id')
    if len(matches):
        sampling_subjects.append(str(matches.iloc[0]['subject_id']))
sampling_subjects = list(dict.fromkeys(sampling_subjects))

patch_rows = []
for subject_offset, subject_id in enumerate(sampling_subjects):
    row = manifest_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    for repeat in range(4):
        center, source = deterministic_patch_center(
            target,
            crop_mask,
            SEED + subject_offset * 100 + repeat,
        )
        slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
        patch_target = extract_patch(
            target, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        patch_rows.append({
            'subject_id': subject_id,
            'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
            'repeat': repeat,
            'center_source': source,
            'full_target_voxels': int(target.sum()),
            'patch_target_voxels': int(patch_target.sum()),
            'positive_patch_contains_lesion': (
                source != 'lesion_jitter' or bool(patch_target.any())
            ),
        })

patch_audit_df = pd.DataFrame(patch_rows)
atomic_write_csv(patch_audit_df, PATCH_AUDIT_PATH)
patch_checks = {
    'all_requested_lesion_bins_sampled': len(sampling_subjects) >= 5,
    'four_patches_per_selected_subject': len(patch_audit_df)
    == 4 * len(sampling_subjects),
    'all_lesion_centered_patches_contain_lesion': bool(
        patch_audit_df['positive_patch_contains_lesion'].all()
    ),
    'empty_subject_sampling_supported': bool(
        (patch_audit_df['full_target_voxels'] == 0).any()
    ),
}
if not all(patch_checks.values()):
    failed = [name for name, passed in patch_checks.items() if not passed]
    raise RuntimeError('Patch-sampling checks failed: ' + ', '.join(failed))

print('\nLesion-aware patch audit:')
display(
    patch_audit_df.groupby(
        ['lesion_volume_bin', 'center_source'], dropna=False
    ).agg(
        patches=('subject_id', 'count'),
        minimum_patch_target_voxels=('patch_target_voxels', 'min'),
        maximum_patch_target_voxels=('patch_target_voxels', 'max'),
    ).reset_index()
)






def soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return (1.0 - (2.0 * intersection + epsilon) / (
        denominator + epsilon
    )).mean()


def boundary_weighted_bce(logits, soft_target, binary_target):
    dilated = F.max_pool3d(binary_target, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool3d(
        -binary_target, kernel_size=3, stride=1, padding=1
    )
    boundary = (dilated - eroded).clamp(0.0, 1.0)
    interpolation_uncertainty = 4.0 * soft_target * (1.0 - soft_target)
    weights = 1.0 + 2.0 * boundary + interpolation_uncertainty
    loss = F.binary_cross_entropy_with_logits(
        logits, soft_target, reduction='none'
    )
    return (loss * weights).sum() / weights.sum().clamp_min(1.0)


def focal_tversky_loss(logits, target, alpha=0.50, beta=0.50, gamma=0.75):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return ((1.0 - tversky).clamp_min(0.0) ** gamma)


def segmentation_loss(logits, soft_target, binary_target, hard_loss_weight):
    dice = soft_dice_loss(logits, soft_target)
    boundary_bce = boundary_weighted_bce(logits, soft_target, binary_target)
    hard_tversky_per_case = focal_tversky_loss(logits, binary_target)
    
    
    hard_loss_weight = hard_loss_weight.reshape(-1, 1)
    hard_tversky = (
        hard_tversky_per_case * hard_loss_weight
    ).sum() / hard_loss_weight.sum().clamp_min(1.0)
    total = 0.55 * dice + 0.30 * boundary_bce + 0.15 * hard_tversky
    return total, {
        'soft_dice': dice,
        'boundary_bce': boundary_bce,
        'hard_focal_tversky': hard_tversky,
    }


def territory_presence_target(binary_target, region_map):
    targets = []
    for label in range(1, NUM_TERRITORY_NODES + 1):
        targets.append(
            ((binary_target > 0) & (region_map == label)).any().astype(np.float32)
        )
    return np.asarray(targets, dtype=np.float32)


eligible_real_df = train_target_df[
    (train_target_df['binary_voxels'] > 0)
    & (train_target_df['binary_loss_weight'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_real_df = eligible_real_df[
    eligible_real_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
candidate_midpoint = len(eligible_real_df) // 2
candidate_order = sorted(
    range(len(eligible_real_df)),
    key=lambda index: (abs(index - candidate_midpoint), index),
)
real_subject = None
for candidate_index in candidate_order:
    candidate_subject = str(
        eligible_real_df.iloc[candidate_index]['subject_id']
    )
    candidate_row = manifest_by_subject[candidate_subject]
    with np.load(
        Path(str(candidate_row['vascufate_v2_npz_path'])),
        allow_pickle=False,
    ) as candidate_data:
        candidate_target = np.asarray(
            candidate_data['target_binary'], dtype=np.uint8
        )
    candidate_region, _ = load_region_map(
        candidate_row, candidate_target.shape
    )
    if territory_presence_target(
        candidate_target, candidate_region
    ).sum() > 0:
        real_subject = candidate_subject
        break
if real_subject is None:
    raise RuntimeError(
        'No atlas-enabled training lesion overlaps a valid territory.'
    )
real_row = manifest_by_subject[real_subject]
real_graph = load_graph_inputs(real_row)
if file_sha256(Path(str(real_row['vascufate_v2_npz_path']))) != str(
    real_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{real_subject}: real-loss NPZ hash changed.')
with np.load(
    Path(str(real_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    real_image = np.asarray(data['image'], dtype=np.float32)
    real_binary = np.asarray(data['target_binary'], dtype=np.float32)
    real_soft = np.asarray(data['target_soft'], dtype=np.float32)
    real_crop_mask = np.asarray(data['crop_mask']) > 0
    real_binary_weight = float(np.asarray(data['binary_loss_weight']).item())
real_region, real_atlas_enabled = load_region_map(
    real_row, real_binary.shape
)
real_territory_target = territory_presence_target(real_binary, real_region)
real_center, real_center_source = deterministic_patch_center(
    real_binary, real_crop_mask, SEED + 9001, positive_probability=1.0
)
real_slices = patch_slices_from_center(
    real_binary.shape, real_center, PATCH_SIZE
)
real_image_patch = extract_patch(
    real_image, real_slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
)
real_binary_patch = extract_patch(
    real_binary, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_soft_patch = extract_patch(
    real_soft, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_region_patch = extract_patch(
    real_region, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)

real_batch = {
    'image': torch.from_numpy(real_image_patch[None]).to(device),
    'region_map': torch.from_numpy(real_region_patch[None]).to(device),
    'territory_features': torch.from_numpy(
        real_graph['territory_features'][None]
    ).to(device),
    'territory_feature_valid': torch.from_numpy(
        real_graph['territory_feature_valid'][None]
    ).to(device),
    'territory_valid': torch.from_numpy(
        real_graph['territory_valid'][None]
    ).to(device),
    'territory_reliability': torch.from_numpy(
        real_graph['territory_reliability'][None]
    ).to(device),
    'vessel_features': torch.from_numpy(
        real_graph['vessel_features'][None]
    ).to(device),
    'vessel_feature_valid': torch.from_numpy(
        real_graph['vessel_feature_valid'][None]
    ).to(device),
    'vessel_valid': torch.from_numpy(real_graph['vessel_valid'][None]).to(device),
    'case_features': torch.from_numpy(real_graph['case_features'][None]).to(device),
    'atlas_enabled': torch.as_tensor(
        [real_atlas_enabled], dtype=torch.bool, device=device
    ),
}
real_soft_tensor = torch.from_numpy(real_soft_patch[None, None]).to(device)
real_binary_tensor = torch.from_numpy(real_binary_patch[None, None]).to(device)
real_hard_weight = torch.as_tensor(
    [real_binary_weight], dtype=torch.float32, device=device
)
real_territory_target_tensor = torch.from_numpy(
    real_territory_target[None]
).to(device)


set_determinism(SEED + 1)
model = VascuFateNet(
    schema=graph_schema,
    image_channels=real_image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
amp_scaler = torch.amp.GradScaler(
    device.type, enabled=device.type == 'cuda'
)
before_step = model.segmentation_head.weight.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
    enabled=device.type == 'cuda',
):
    real_output = model(**real_batch, force_atlas_dropout=False)
    main_loss, main_parts = segmentation_loss(
        real_output['logits'],
        real_soft_tensor,
        real_binary_tensor,
        real_hard_weight,
    )
    half_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        real_output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        real_hard_weight,
    )
    quarter_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        real_output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        real_hard_weight,
    )
    node_mask = (
        real_batch['territory_valid'].bool()
        & real_output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        real_output['territory_node_logits'],
        real_territory_target_tensor,
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = real_output['territory_node_logits'].sum() * 0.0
    total_real_loss = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )

amp_scaler.scale(total_real_loss).backward()
amp_scaler.unscale_(optimizer)
real_gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0).item()
)
amp_scaler.step(optimizer)
amp_scaler.update()
optimizer.zero_grad(set_to_none=True)
parameter_update = float(
    (model.segmentation_head.weight.detach() - before_step).abs().max().item()
)

loss_values = {
    'subject_id': real_subject,
    'patch_center_source': real_center_source,
    'patch_target_voxels': int(real_binary_patch.sum()),
    'main_loss': float(main_loss.detach().cpu()),
    'soft_dice_loss': float(main_parts['soft_dice'].detach().cpu()),
    'boundary_bce_loss': float(main_parts['boundary_bce'].detach().cpu()),
    'hard_focal_tversky_loss': float(
        main_parts['hard_focal_tversky'].detach().cpu()
    ),
    'half_resolution_loss': float(half_loss.detach().cpu()),
    'quarter_resolution_loss': float(quarter_loss.detach().cpu()),
    'territory_node_loss': float(node_loss.detach().cpu()),
    'total_loss': float(total_real_loss.detach().cpu()),
    'gradient_norm_before_clipping': real_gradient_norm,
    'segmentation_head_maximum_update': parameter_update,
}
atomic_write_json(loss_values, LOSS_AUDIT_PATH)
loss_checks = {
    'real_patch_contains_lesion': int(real_binary_patch.sum()) > 0,
    'all_real_loss_components_finite': all(
        math.isfinite(float(value))
        for key, value in loss_values.items()
        if key.endswith('_loss') or key == 'total_loss'
    ),
    'real_gradient_norm_positive_and_finite': math.isfinite(real_gradient_norm)
    and real_gradient_norm > 0,
    'optimizer_step_changed_parameters': parameter_update > 0,
    'territory_target_has_30_nodes': real_territory_target.shape == (30,),
    'territory_target_contains_positive_node': real_territory_target.sum() > 0,
    'parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
}
if not all(loss_checks.values()):
    failed = [name for name, passed in loss_checks.items() if not passed]
    raise RuntimeError('Real-loss optimizer preflight failed: ' + ', '.join(failed))

print('\nReal-loss disposable optimizer-step audit:')
display(pd.DataFrame([
    {'metric': name, 'value': value} for name, value in loss_values.items()
]))
print('\nParameter count after model reinitialization:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': parameter_count(model)},
    {'component': 'trainable', 'parameters': parameter_count(model, True)},
]))






def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


@torch.no_grad()
def sliding_window_probability(model, row, image, region_map, graph_inputs):
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, PATCH_SIZE)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(PATCH_SIZE)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, PATCH_SIZE)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    model.eval()
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index : location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(
                np.repeat(graph_inputs['territory_features'][None], count, axis=0)
            ).to(device),
            'territory_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'territory_valid': torch.from_numpy(
                np.repeat(graph_inputs['territory_valid'][None], count, axis=0)
            ).to(device),
            'territory_reliability': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_reliability'][None], count, axis=0
                )
            ).to(device),
            'vessel_features': torch.from_numpy(
                np.repeat(graph_inputs['vessel_features'][None], count, axis=0)
            ).to(device),
            'vessel_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['vessel_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'vessel_valid': torch.from_numpy(
                np.repeat(graph_inputs['vessel_valid'][None], count, axis=0)
            ).to(device),
            'case_features': torch.from_numpy(
                np.repeat(graph_inputs['case_features'][None], count, axis=0)
            ).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **inference_batch, force_atlas_dropout=False
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    return probability[original_slices]


eligible_validation_df = target_audit_df[
    (target_audit_df['dev_split'] == 'val')
    & (target_audit_df['binary_voxels'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_validation_df = eligible_validation_df[
    eligible_validation_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
validation_subject = str(
    eligible_validation_df.iloc[len(eligible_validation_df) // 2]['subject_id']
)
validation_row = manifest_by_subject[validation_subject]
validation_graph = load_graph_inputs(validation_row)
if file_sha256(Path(str(validation_row['vascufate_v2_npz_path']))) != str(
    validation_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{validation_subject}: validation NPZ hash changed.')
with np.load(
    Path(str(validation_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    validation_image = np.asarray(data['image'], dtype=np.float32)
    validation_target = np.asarray(data['target_binary'], dtype=np.uint8)
validation_region, _ = load_region_map(
    validation_row, validation_target.shape
)
validation_probability = sliding_window_probability(
    model,
    validation_row,
    validation_image,
    validation_region,
    validation_graph,
)
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    prediction = validation_probability >= threshold
    intersection = int((prediction & (validation_target > 0)).sum())
    predicted = int(prediction.sum())
    truth = int(validation_target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else (2.0 * intersection) / (predicted + truth)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    precision = 1.0 if predicted == 0 and truth == 0 else (
        intersection / max(predicted, 1)
    )
    threshold_rows.append({
        'subject_id': validation_subject,
        'threshold': threshold,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    })
validation_metric_df = pd.DataFrame(threshold_rows)
atomic_write_csv(validation_metric_df, VALIDATION_AUDIT_PATH)
validation_checks = {
    'validation_probability_shape_matches': validation_probability.shape
    == validation_target.shape,
    'validation_probability_finite': bool(
        np.isfinite(validation_probability).all()
    ),
    'validation_probability_in_unit_interval': bool(
        validation_probability.min() >= 0
        and validation_probability.max() <= 1
    ),
    'five_thresholds_evaluated': len(validation_metric_df) == 5,
    'validation_metrics_finite': bool(
        np.isfinite(
            validation_metric_df[['dice', 'precision', 'recall']].to_numpy()
        ).all()
    ),
    'test_targets_still_not_opened': target_arrays_read_splits == {
        'train',
        'val',
    },
}
if not all(validation_checks.values()):
    failed = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Sliding-window validation preflight failed: ' + ', '.join(failed)
    )

print('\nDisposable full-volume validation smoke metrics:')
display(validation_metric_df)






training_policy = {
    'patch_size': list(PATCH_SIZE),
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'patches_per_subject_per_epoch': 2,
    'sampling_schedule': 'one_lesion_centered_plus_one_brain_context',
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'early_stopping_patience_validations': 10,
    'optimizer': 'AdamW',
    'learning_rate': 1e-4,
    'weight_decay': 5e-4,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'positive_center_probability': 1.0,
    'context_center_probability': 1.0,
    'intensity_gain_range': [0.90, 1.10],
    'intensity_offset_range': [-0.03, 0.03],
    'noise_probability': 0.65,
    'maximum_noise_standard_deviation': 0.02,
    'gamma_probability': 0.50,
    'gamma_range': [0.85, 1.15],
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'main_loss_weights': {
        'soft_dice': 0.55,
        'boundary_weighted_bce': 0.30,
        'hard_focal_tversky': 0.15,
    },
    'focal_tversky_alpha_false_positive': 0.50,
    'focal_tversky_beta_false_negative': 0.50,
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.05,
    'territory_positive_weight': 4.0,
    'validation_inference': '50_percent_overlap_gaussian_sliding_window',
    'threshold_candidates': [
        round(value, 2) for value in np.arange(0.10, 0.61, 0.05)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_training_checks = {
    **frozen_architecture_checks,
    **target_audit_checks,
    **patch_checks,
    **loss_checks,
    **validation_checks,
    'frozen_scaler_hash_matches': scaler_sha256 == EXPECTED_SCALER_SHA256,
    'target_free_model_recheck_passed': preflight_passed,
    'total_parameters_match': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameters_match': parameter_count(model, True)
    == EXPECTED_TOTAL_PARAMETERS,
    'test_split_remains_sealed': 'test' not in target_arrays_read_splits,
}
training_preflight_passed = bool(all(all_training_checks.values()))
atomic_write_csv(
    pd.DataFrame([
        {'check': name, 'passed': passed}
        for name, passed in all_training_checks.items()
    ]),
    PREFLIGHT_AUDIT_PATH,
)
final_gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'training_preflight_passed': training_preflight_passed,
    'full_training_authorized': training_preflight_passed,
    'checks': all_training_checks,
    'training_policy': training_policy,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_sha256': architecture_sha256,
    'total_parameters': parameter_count(model),
    'trainable_parameters': parameter_count(model, True),
    'real_loss_subject': real_subject,
    'validation_smoke_subject': validation_subject,
    'target_arrays_read_splits': sorted(target_arrays_read_splits),
    'test_targets_read': False,
    'study_training_epochs_completed': 0,
}
atomic_write_json(final_gate, PREFLIGHT_GATE_PATH)

print('\nFinal target-aware training preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_training_checks.items()
]))
if not training_preflight_passed:
    failed = [
        name for name, passed in all_training_checks.items() if not passed
    ]
    raise RuntimeError(
        'TRIAL8 TARGET-AWARE TRAINING PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT PASSED')
print('=' * 80)
print('Train targets audited:              102')
print('Validation targets audited:         23')
print('Test targets read:                  False')
print(f'Real-loss subject:                  {real_subject}')
print(f'Validation smoke subject:           {validation_subject}')
print(f'Total parameters:                   {parameter_count(model):,}')
print(f'Trainable parameters:               {parameter_count(model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Frozen scaler SHA-256:              {scaler_sha256}')
print(f'Training preflight gate:            {PREFLIGHT_GATE_PATH}')
print('Study training epochs completed:    0')
print('Full Trial8 training authorized:    True')
print('\nTarget-aware gate passed; starting checkpoint diagnostic below.')








del model, optimizer, amp_scaler, real_output, real_batch
del real_soft_tensor, real_binary_tensor, real_territory_target_tensor
del normal_output, dropped_output, gradient_output, batch
del smoke_loss, total_real_loss, main_loss, half_loss, quarter_loss
del node_loss, node_bce, before_step
del validation_probability, validation_image, validation_target
del validation_region, validation_graph
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DIAGNOSTIC_VERSION = 'v1_dense_threshold_v1_v2_probability_ensemble'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
V1_BEST_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / 'v1_ragatv2_film_boundary_region'
    / 'best_checkpoint.pt'
)
V2_BEST_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / 'v2_regularized_balanced_epochwise'
    / 'best_checkpoint.pt'
)
DIAGNOSTIC_ROOT = CHECKPOINT_ROOT / DIAGNOSTIC_VERSION
CASE_METRICS_PATH = DIAGNOSTIC_ROOT / 'validation_case_metrics.csv'
CONFIGURATION_SUMMARY_PATH = (
    DIAGNOSTIC_ROOT / 'validation_configuration_summary.csv'
)
SELECTED_CONFIGURATIONS_PATH = (
    DIAGNOSTIC_ROOT / 'selected_configurations.csv'
)
LESION_BIN_SUMMARY_PATH = (
    DIAGNOSTIC_ROOT / 'selected_lesion_bin_summary.csv'
)
PAIRED_BOOTSTRAP_PATH = (
    DIAGNOSTIC_ROOT / 'paired_bootstrap_differences.csv'
)
DIAGNOSTIC_GATE_PATH = DIAGNOSTIC_ROOT / 'diagnostic_gate.json'

for path in [
    V1_BEST_CHECKPOINT_PATH,
    V2_BEST_CHECKPOINT_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            'Required best checkpoint is missing: ' + str(path)
        )
assert_project_path_is_safe(DIAGNOSTIC_ROOT)
DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)



DENSE_THRESHOLDS = np.round(
    np.arange(0.05, 0.9501, 0.025), 3
).astype(np.float64)
V2_ENSEMBLE_WEIGHTS = np.asarray(
    [0.00, 0.25, 0.50, 0.75, 1.00], dtype=np.float64
)
V2_REPORTED_THRESHOLD = 0.60
BOOTSTRAP_REPLICATES = 10_000


def build_study_model():
    study_model = VascuFateNet(
        schema=graph_schema,
        image_channels=15,
        territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
        vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
        case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
        graph_hidden=GRAPH_HIDDEN,
        graph_heads=GRAPH_HEADS,
        graph_layers=GRAPH_LAYERS,
        relation_bases=RELATION_BASES,
        graph_dropout=GRAPH_DROPOUT,
        atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
        film_bound=FILM_BOUND,
    ).to(device)
    if parameter_count(study_model) != EXPECTED_TOTAL_PARAMETERS:
        raise RuntimeError('Diagnostic model parameter count changed.')
    return study_model


def load_read_only_checkpoint(path, label):
    checkpoint = torch.load(
        path, map_location='cpu', weights_only=False
    )
    checks = {
        'model_state_dict_present': 'model_state_dict' in checkpoint,
        'manifest_hash_matches': checkpoint.get(
            'verified_manifest_sha256'
        ) == EXPECTED_VERIFIED_MANIFEST_SHA256,
        'scaler_hash_matches': checkpoint.get(
            'scaler_sha256'
        ) == EXPECTED_SCALER_SHA256,
        'graph_schema_hash_matches': checkpoint.get(
            'graph_schema_sha256'
        ) == EXPECTED_SAFE_SCHEMA_SHA256,
        'parameter_count_matches': checkpoint.get(
            'total_parameters'
        ) == EXPECTED_TOTAL_PARAMETERS,
        'test_targets_were_not_read': checkpoint.get(
            'test_targets_read'
        ) is False,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'{label} checkpoint integrity failed: ' + ', '.join(failed)
        )
    study_model = build_study_model()
    study_model.load_state_dict(
        checkpoint['model_state_dict'], strict=True
    )
    study_model.eval()
    metadata = {
        'checkpoint_label': label,
        'checkpoint_path': str(path),
        'checkpoint_sha256': file_sha256(path),
        'training_version': str(checkpoint.get('training_version', '')),
        'best_epoch': int(checkpoint.get('best_epoch', checkpoint['epoch'])),
        'reported_best_mean_dice': float(
            checkpoint.get('best_mean_dice', np.nan)
        ),
        'reported_best_threshold': float(
            checkpoint.get('best_threshold', np.nan)
        ),
        'total_parameters': parameter_count(study_model),
        'trainable_parameters': parameter_count(study_model, True),
    }
    return study_model, metadata


def threshold_counts(probability, target, thresholds):
    probability = np.asarray(probability, dtype=np.float32).reshape(-1)
    target = np.asarray(target, dtype=bool).reshape(-1)
    if probability.shape != target.shape:
        raise RuntimeError('Probability and target shapes differ.')
    if not np.isfinite(probability).all():
        raise RuntimeError('Non-finite validation probabilities found.')
    if float(probability.min()) < 0.0 or float(probability.max()) > 1.0:
        raise RuntimeError('Validation probabilities are outside [0, 1].')
    
    
    edges = np.concatenate((
        np.asarray([-np.inf], dtype=np.float32),
        np.asarray(thresholds, dtype=np.float32),
        np.asarray([np.inf], dtype=np.float32),
    ))
    positive_histogram = np.histogram(
        probability[target], bins=edges
    )[0].astype(np.int64)
    negative_histogram = np.histogram(
        probability[~target], bins=edges
    )[0].astype(np.int64)
    true_positive = np.cumsum(positive_histogram[::-1])[::-1][1:]
    false_positive = np.cumsum(negative_histogram[::-1])[::-1][1:]
    truth = int(target.sum())
    predicted = true_positive + false_positive
    false_negative = truth - true_positive
    denominator = predicted + truth
    dice = np.divide(
        2.0 * true_positive,
        denominator,
        out=np.ones_like(denominator, dtype=np.float64),
        where=denominator > 0,
    )
    precision = np.divide(
        true_positive,
        predicted,
        out=np.where(truth == 0, 1.0, 0.0).astype(np.float64)
        * np.ones_like(predicted, dtype=np.float64),
        where=predicted > 0,
    )
    recall = (
        np.ones_like(true_positive, dtype=np.float64)
        if truth == 0
        else true_positive.astype(np.float64) / truth
    )
    return {
        'intersection': true_positive,
        'false_positive_voxels': false_positive,
        'false_negative_voxels': false_negative,
        'predicted_voxels': predicted,
        'target_voxels': np.full_like(predicted, truth),
        'dice': dice,
        'precision': precision,
        'recall': recall,
    }


def summarize_configurations(case_frame):
    grouped = (
        case_frame.groupby(['v2_weight', 'threshold'], as_index=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_subject_dice=('dice', 'mean'),
            median_subject_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_error_ml=(
                'absolute_volume_error_ml', 'mean'
            ),
            total_intersection=('intersection', 'sum'),
            total_predicted_voxels=('predicted_voxels', 'sum'),
            total_target_voxels=('target_voxels', 'sum'),
        )
    )
    grouped['global_dice'] = (
        2.0 * grouped['total_intersection']
        / np.maximum(
            grouped['total_predicted_voxels']
            + grouped['total_target_voxels'],
            1,
        )
    )
    grouped['model'] = grouped['v2_weight'].map(
        lambda value: (
            'v1'
            if np.isclose(value, 0.0)
            else 'v2'
            if np.isclose(value, 1.0)
            else f'v1_v2_ensemble_v2_weight_{value:.2f}'
        )
    )
    return grouped.sort_values(
        [
            'mean_subject_dice',
            'median_subject_dice',
            'mean_recall',
            'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)


def select_best(summary, weights=None):
    candidates = summary
    if weights is not None:
        candidates = candidates[
            candidates['v2_weight'].map(
                lambda value: any(np.isclose(value, item) for item in weights)
            )
        ]
    if candidates.empty:
        raise RuntimeError('No diagnostic configuration can be selected.')
    return candidates.iloc[0].to_dict()


def configuration_cases(case_frame, configuration):
    return case_frame[
        np.isclose(case_frame['v2_weight'], configuration['v2_weight'])
        & np.isclose(case_frame['threshold'], configuration['threshold'])
    ].sort_values('subject_id').reset_index(drop=True)


def paired_bootstrap_difference(reference, candidate, label, seed):
    merged = reference[
        ['subject_id', 'dice']
    ].merge(
        candidate[['subject_id', 'dice']],
        on='subject_id',
        suffixes=('_reference', '_candidate'),
        validate='one_to_one',
    )
    differences = (
        merged['dice_candidate'] - merged['dice_reference']
    ).to_numpy(dtype=np.float64)
    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(
        0,
        len(differences),
        size=(BOOTSTRAP_REPLICATES, len(differences)),
    )
    bootstrap_means = differences[sample_indices].mean(axis=1)
    return {
        'comparison': label,
        'subjects': len(differences),
        'mean_dice_difference': float(differences.mean()),
        'median_dice_difference': float(np.median(differences)),
        'subjects_improved': int((differences > 0).sum()),
        'subjects_tied': int(np.isclose(differences, 0.0).sum()),
        'subjects_worsened': int((differences < 0).sum()),
        'descriptive_bootstrap_ci95_low': float(
            np.quantile(bootstrap_means, 0.025)
        ),
        'descriptive_bootstrap_ci95_high': float(
            np.quantile(bootstrap_means, 0.975)
        ),
        'note': (
            'Descriptive only: configurations were selected on this same '
            'validation split.'
        ),
    }


v1_model, v1_metadata = load_read_only_checkpoint(
    V1_BEST_CHECKPOINT_PATH, 'v1_baseline_best'
)
v2_model, v2_metadata = load_read_only_checkpoint(
    V2_BEST_CHECKPOINT_PATH, 'v2_regularized_best'
)

checkpoint_metadata_df = pd.DataFrame([v1_metadata, v2_metadata])
print('\nRead-only checkpoint metadata:')
display(checkpoint_metadata_df)

validation_rows = (
    manifest_df[manifest_df['dev_split'] == 'val']
    .sort_values('subject_id')
    .to_dict('records')
)
if len(validation_rows) != EXPECTED_SPLITS['val']:
    raise RuntimeError('Expected exactly 23 validation subjects.')

case_rows = []
for case_index, row in enumerate(validation_rows, start=1):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: validation NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
    region_map, _ = load_region_map(row, target.shape)
    graph_inputs = load_graph_inputs(row)
    v1_probability = sliding_window_probability(
        v1_model, row, image, region_map, graph_inputs
    )
    v2_probability = sliding_window_probability(
        v2_model, row, image, region_map, graph_inputs
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    lesion_bin = str(row.get('lesion_volume_bin', 'unknown'))
    for v2_weight in V2_ENSEMBLE_WEIGHTS:
        probability = (
            (1.0 - float(v2_weight)) * v1_probability
            + float(v2_weight) * v2_probability
        )
        metric_arrays = threshold_counts(
            probability, target > 0, DENSE_THRESHOLDS
        )
        for threshold_index, threshold in enumerate(DENSE_THRESHOLDS):
            predicted_voxels = int(
                metric_arrays['predicted_voxels'][threshold_index]
            )
            target_voxels = int(
                metric_arrays['target_voxels'][threshold_index]
            )
            case_rows.append({
                'subject_id': subject_id,
                'lesion_volume_bin': lesion_bin,
                'v2_weight': float(v2_weight),
                'v1_weight': float(1.0 - v2_weight),
                'threshold': float(threshold),
                'dice': float(metric_arrays['dice'][threshold_index]),
                'precision': float(
                    metric_arrays['precision'][threshold_index]
                ),
                'recall': float(metric_arrays['recall'][threshold_index]),
                'intersection': int(
                    metric_arrays['intersection'][threshold_index]
                ),
                'false_positive_voxels': int(
                    metric_arrays['false_positive_voxels'][threshold_index]
                ),
                'false_negative_voxels': int(
                    metric_arrays['false_negative_voxels'][threshold_index]
                ),
                'predicted_voxels': predicted_voxels,
                'target_voxels': target_voxels,
                'predicted_volume_ml': predicted_voxels * voxel_volume_ml,
                'target_volume_ml': target_voxels * voxel_volume_ml,
                'absolute_volume_error_ml': (
                    abs(predicted_voxels - target_voxels) * voxel_volume_ml
                ),
                'target_arrays_read_from_split': 'val',
            })
    del image, target, region_map, graph_inputs
    del v1_probability, v2_probability
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if case_index == 1 or case_index % 5 == 0 or case_index == len(validation_rows):
        print(
            f'Evaluated {case_index}/{len(validation_rows)}: {subject_id}'
        )

case_metrics_df = pd.DataFrame(case_rows)
configuration_summary_df = summarize_configurations(case_metrics_df)

v1_best = select_best(configuration_summary_df, weights=[0.0])
v2_dense_best = select_best(configuration_summary_df, weights=[1.0])
ensemble_best = select_best(
    configuration_summary_df, weights=[0.25, 0.50, 0.75]
)
overall_best = select_best(configuration_summary_df)
v2_reported_row = configuration_summary_df[
    np.isclose(configuration_summary_df['v2_weight'], 1.0)
    & np.isclose(
        configuration_summary_df['threshold'], V2_REPORTED_THRESHOLD
    )
]
if len(v2_reported_row) != 1:
    raise RuntimeError('Could not recover the v2 threshold-0.60 baseline.')
v2_reported = v2_reported_row.iloc[0].to_dict()

selected_rows = []
for selection_name, record in [
    ('v2_reported_threshold', v2_reported),
    ('v1_dense_best', v1_best),
    ('v2_dense_best', v2_dense_best),
    ('ensemble_dense_best', ensemble_best),
    ('overall_dense_best', overall_best),
]:
    selected_rows.append({'selection': selection_name, **record})
selected_configurations_df = pd.DataFrame(selected_rows)

lesion_bin_rows = []
selected_case_frames = {}
for record in selected_rows:
    selection = str(record['selection'])
    selected_cases = configuration_cases(case_metrics_df, record)
    selected_case_frames[selection] = selected_cases
    grouped = (
        selected_cases.groupby('lesion_volume_bin', dropna=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_dice=('dice', 'mean'),
            median_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
        )
        .reset_index()
    )
    grouped.insert(0, 'selection', selection)
    lesion_bin_rows.extend(grouped.to_dict('records'))
lesion_bin_summary_df = pd.DataFrame(lesion_bin_rows)

bootstrap_rows = [
    paired_bootstrap_difference(
        selected_case_frames['v2_reported_threshold'],
        selected_case_frames['v2_dense_best'],
        'v2_dense_minus_v2_reported',
        SEED + 101,
    ),
    paired_bootstrap_difference(
        selected_case_frames['v2_reported_threshold'],
        selected_case_frames['ensemble_dense_best'],
        'ensemble_dense_minus_v2_reported',
        SEED + 102,
    ),
    paired_bootstrap_difference(
        selected_case_frames['v2_dense_best'],
        selected_case_frames['ensemble_dense_best'],
        'ensemble_dense_minus_v2_dense',
        SEED + 103,
    ),
]
paired_bootstrap_df = pd.DataFrame(bootstrap_rows)

diagnostic_checks = {
    'exactly_23_validation_subjects': (
        case_metrics_df['subject_id'].nunique() == 23
    ),
    'only_validation_targets_read': set(
        case_metrics_df['target_arrays_read_from_split'].unique()
    ) == {'val'},
    'test_targets_not_read': 'test' not in target_arrays_read_splits,
    'five_checkpoint_weight_settings_evaluated': (
        case_metrics_df['v2_weight'].nunique() == 5
    ),
    'thirty_seven_dense_thresholds_evaluated': (
        case_metrics_df['threshold'].nunique() == 37
    ),
    'all_metrics_finite': bool(np.isfinite(
        case_metrics_df[
            ['dice', 'precision', 'recall', 'absolute_volume_error_ml']
        ].to_numpy()
    ).all()),
    'all_probabilistic_metrics_bounded': bool(
        case_metrics_df[['dice', 'precision', 'recall']]
        .ge(0.0).all().all()
        and case_metrics_df[['dice', 'precision', 'recall']]
        .le(1.0).all().all()
    ),
    'source_checkpoints_unchanged': (
        file_sha256(V1_BEST_CHECKPOINT_PATH)
        == v1_metadata['checkpoint_sha256']
        and file_sha256(V2_BEST_CHECKPOINT_PATH)
        == v2_metadata['checkpoint_sha256']
    ),
    'architecture_parameter_count_unchanged': (
        parameter_count(v1_model) == EXPECTED_TOTAL_PARAMETERS
        and parameter_count(v2_model) == EXPECTED_TOTAL_PARAMETERS
    ),
    'scientific_core_unchanged': True,
    'no_training_performed': True,
}
diagnostic_passed = bool(all(diagnostic_checks.values()))

atomic_write_csv(case_metrics_df, CASE_METRICS_PATH)
atomic_write_csv(
    configuration_summary_df, CONFIGURATION_SUMMARY_PATH
)
atomic_write_csv(
    selected_configurations_df, SELECTED_CONFIGURATIONS_PATH
)
atomic_write_csv(lesion_bin_summary_df, LESION_BIN_SUMMARY_PATH)
atomic_write_csv(paired_bootstrap_df, PAIRED_BOOTSTRAP_PATH)

best_threshold_at_boundary = bool(
    np.isclose(overall_best['threshold'], DENSE_THRESHOLDS.min())
    or np.isclose(overall_best['threshold'], DENSE_THRESHOLDS.max())
)
atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': DIAGNOSTIC_VERSION,
        'diagnostic_passed': diagnostic_passed,
        'checks': diagnostic_checks,
        'validation_subjects': 23,
        'test_targets_read': False,
        'training_performed': False,
        'study_trainable_parameters': 0,
        'model_parameters_per_checkpoint': EXPECTED_TOTAL_PARAMETERS,
        'scientific_core': (
            '15-channel residual 3D U-Net plus heterogeneous relation-aware '
            'GATv2 with bounded zero-init multiscale FiLM'
        ),
        'scientific_core_changed': False,
        'v1_checkpoint': v1_metadata,
        'v2_checkpoint': v2_metadata,
        'dense_thresholds': DENSE_THRESHOLDS.tolist(),
        'v2_ensemble_weights': V2_ENSEMBLE_WEIGHTS.tolist(),
        'selected_configurations': [
            to_builtin(record) for record in selected_rows
        ],
        'overall_best_threshold_at_search_boundary': (
            best_threshold_at_boundary
        ),
        'v3_training_authorized': False,
        'next_action': (
            'Check calibration and ensemble evidence before freezing the '
            'v3 imbalance-aware training policy.'
        ),
    },
    DIAGNOSTIC_GATE_PATH,
)

print('\nDense-threshold and ensemble selections:')
display(selected_configurations_df[[
    'selection',
    'model',
    'v1_weight',
    'v2_weight',
    'threshold',
    'mean_subject_dice',
    'median_subject_dice',
    'global_dice',
    'mean_precision',
    'mean_recall',
    'mean_absolute_volume_error_ml',
]])

print('\nSelected performance by lesion-volume bin:')
display(lesion_bin_summary_df)

print('\nPaired validation differences:')
display(paired_bootstrap_df)

print('\nDiagnostic checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in diagnostic_checks.items()
]))
if not diagnostic_passed:
    failed = [
        name for name, passed in diagnostic_checks.items() if not passed
    ]
    raise RuntimeError(
        'TRIAL8 THRESHOLD/ENSEMBLE DIAGNOSTIC FAILED. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 DENSE-THRESHOLD AND ENSEMBLE DIAGNOSTIC PASSED')
print('=' * 80)
print('Validation subjects:                23')
print('Test targets read:                  False')
print('Training performed:                 False')
print('Study trainable parameters:         0 (diagnostic only)')
print(f'Model parameters per checkpoint:    {EXPECTED_TOTAL_PARAMETERS:,}')
print('Architecture/scientific core changed: False')
print(f'V1 dense best mean Dice:            {v1_best["mean_subject_dice"]:.4f}')
print(f'V2 reported mean Dice:              {v2_reported["mean_subject_dice"]:.4f}')
print(f'V2 dense best mean Dice:            {v2_dense_best["mean_subject_dice"]:.4f}')
print(f'Ensemble dense best mean Dice:      {ensemble_best["mean_subject_dice"]:.4f}')
print(f'Overall best model:                 {overall_best["model"]}')
print(f'Overall best threshold:             {overall_best["threshold"]:.3f}')
print(f'Overall best mean Dice:             {overall_best["mean_subject_dice"]:.4f}')
print(f'Overall threshold at boundary:      {best_threshold_at_boundary}')
print('V3 training authorized:             False')
print(f'Case metrics:                       {CASE_METRICS_PATH}')
print(f'Configuration summary:              {CONFIGURATION_SUMMARY_PATH}')
print(f'Diagnostic gate:                    {DIAGNOSTIC_GATE_PATH}')
print('\nNo source checkpoint was modified.')
print('Stop here and check the calibration and ensemble evidence before v3.')


In [ ]:
import sys
import os
import gc
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v2_regularized_epochwise_validation'
CODE_REVISION = 'r2_overfit_control_epochwise_validation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')






print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT')
print('=' * 80)
print('Target access policy: train and validation only; test remains sealed.')

target_rows = []
target_arrays_read_splits = set()
for row in manifest_df[
    manifest_df['dev_split'].isin(['train', 'val'])
].sort_values(['dev_split', 'subject_id']).to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    source_path = Path(str(row['vascufate_v2_npz_path']))
    if not source_path.exists():
        raise FileNotFoundError(f'{subject_id}: frozen NPZ is missing.')
    with np.load(source_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'binary_loss_weight',
            'crop_mask',
            'target_spacing',
            'subject_id',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        spatial_shape = tuple(int(value) for value in target_binary.shape)
        crop_mask = np.asarray(data['crop_mask']) > 0
        binary_loss_weight_value = float(
            np.asarray(data['binary_loss_weight']).item()
        )
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        stored_subject = scalar_string(data['subject_id'])
    target_arrays_read_splits.add(split)
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: stored subject identifier mismatch.')
    if target_soft.shape != target_binary.shape or crop_mask.shape != target_binary.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not set(int(value) for value in np.unique(target_binary)).issubset({0, 1}):
        raise RuntimeError(f'{subject_id}: binary target is not binary.')
    if (
        not np.isfinite(target_soft).all()
        or target_soft.min() < -1e-4
        or target_soft.max() > 1.0001
    ):
        raise RuntimeError(f'{subject_id}: soft target is invalid.')
    if binary_loss_weight_value not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    target_voxels = int(target_binary.sum())
    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
        'spatial_shape': str(spatial_shape),
        'binary_voxels': target_voxels,
        'binary_volume_ml': float(target_voxels * voxel_volume_ml),
        'soft_sum': float(target_soft.sum(dtype=np.float64)),
        'binary_loss_weight': binary_loss_weight_value,
        'target_inside_crop_fraction': 1.0 if target_voxels == 0 else float(
            (target_binary & crop_mask).sum() / target_voxels
        ),
        'all_finite': True,
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_audit_checks = {
    'exactly_102_train_targets_audited': int(
        (target_audit_df['dev_split'] == 'train').sum()
    )
    == 102,
    'exactly_23_validation_targets_audited': int(
        (target_audit_df['dev_split'] == 'val').sum()
    )
    == 23,
    'test_targets_not_opened': target_arrays_read_splits == {'train', 'val'},
    'all_target_arrays_finite': bool(target_audit_df['all_finite'].all()),
    'target_crop_support_fractions_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'train_contains_nonempty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] > 0)
        ).any()
    ),
    'train_contains_empty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] == 0)
        ).any()
    ),
}
if not all(target_audit_checks.values()):
    failed = [
        name for name, passed in target_audit_checks.items() if not passed
    ]
    raise RuntimeError('Target audit failed: ' + ', '.join(failed))

print('\nTarget audit by split:')
display(
    target_audit_df.groupby('dev_split').agg(
        subjects=('subject_id', 'count'),
        empty_targets=('binary_voxels', lambda values: int((values == 0).sum())),
        minimum_volume_ml=('binary_volume_ml', 'min'),
        median_volume_ml=('binary_volume_ml', 'median'),
        maximum_volume_ml=('binary_volume_ml', 'max'),
    ).reset_index()
)
print('\nTarget audit by training lesion bin:')
display(
    target_audit_df[target_audit_df['dev_split'] == 'train']
    .groupby('lesion_volume_bin')
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('binary_volume_ml', 'median'),
    )
    .reset_index()
)






def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)

    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(f'{row["subject_id"]}: atlas labels outside 0..30.')
    return labels, True


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size, spatial_axes):
    source_slices = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source_slices[axis] = spatial_slice
    cropped = array[tuple(source_slices)]
    return center_crop_or_pad(cropped, patch_size, spatial_axes)


def deterministic_patch_center(target, crop_mask, seed, positive_probability=0.75):
    rng = np.random.default_rng(seed)
    positive_coordinates = np.argwhere(target > 0)
    use_positive = (
        len(positive_coordinates) > 0
        and float(rng.random()) < positive_probability
    )
    if use_positive:
        center = positive_coordinates[
            int(rng.integers(0, len(positive_coordinates)))
        ].astype(np.int64)
        jitter = np.asarray([
            int(rng.integers(-16, 17)),
            int(rng.integers(-16, 17)),
            int(rng.integers(-6, 7)),
        ])
        center = center + jitter
        source = 'lesion_jitter'
    else:
        candidates = np.argwhere(crop_mask > 0)
        if len(candidates) == 0:
            center = np.asarray(target.shape) // 2
        else:
            center = candidates[int(rng.integers(0, len(candidates)))]
        source = 'brain_random'
    center = np.clip(center, 0, np.asarray(target.shape) - 1)
    return center, source


manifest_by_subject = {
    str(row['subject_id']): row
    for row in manifest_df.to_dict('records')
}
train_target_df = target_audit_df[target_audit_df['dev_split'] == 'train']
sampling_subjects = []
for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
    matches = train_target_df[
        train_target_df['lesion_volume_bin'] == lesion_bin
    ].sort_values('subject_id')
    if len(matches):
        sampling_subjects.append(str(matches.iloc[0]['subject_id']))
sampling_subjects = list(dict.fromkeys(sampling_subjects))

patch_rows = []
for subject_offset, subject_id in enumerate(sampling_subjects):
    row = manifest_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    for repeat in range(4):
        center, source = deterministic_patch_center(
            target,
            crop_mask,
            SEED + subject_offset * 100 + repeat,
        )
        slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
        patch_target = extract_patch(
            target, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        patch_rows.append({
            'subject_id': subject_id,
            'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
            'repeat': repeat,
            'center_source': source,
            'full_target_voxels': int(target.sum()),
            'patch_target_voxels': int(patch_target.sum()),
            'positive_patch_contains_lesion': (
                source != 'lesion_jitter' or bool(patch_target.any())
            ),
        })

patch_audit_df = pd.DataFrame(patch_rows)
atomic_write_csv(patch_audit_df, PATCH_AUDIT_PATH)
patch_checks = {
    'all_requested_lesion_bins_sampled': len(sampling_subjects) >= 5,
    'four_patches_per_selected_subject': len(patch_audit_df)
    == 4 * len(sampling_subjects),
    'all_lesion_centered_patches_contain_lesion': bool(
        patch_audit_df['positive_patch_contains_lesion'].all()
    ),
    'empty_subject_sampling_supported': bool(
        (patch_audit_df['full_target_voxels'] == 0).any()
    ),
}
if not all(patch_checks.values()):
    failed = [name for name, passed in patch_checks.items() if not passed]
    raise RuntimeError('Patch-sampling checks failed: ' + ', '.join(failed))

print('\nLesion-aware patch audit:')
display(
    patch_audit_df.groupby(
        ['lesion_volume_bin', 'center_source'], dropna=False
    ).agg(
        patches=('subject_id', 'count'),
        minimum_patch_target_voxels=('patch_target_voxels', 'min'),
        maximum_patch_target_voxels=('patch_target_voxels', 'max'),
    ).reset_index()
)






def soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return (1.0 - (2.0 * intersection + epsilon) / (
        denominator + epsilon
    )).mean()


def boundary_weighted_bce(logits, soft_target, binary_target):
    dilated = F.max_pool3d(binary_target, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool3d(
        -binary_target, kernel_size=3, stride=1, padding=1
    )
    boundary = (dilated - eroded).clamp(0.0, 1.0)
    interpolation_uncertainty = 4.0 * soft_target * (1.0 - soft_target)
    weights = 1.0 + 2.0 * boundary + interpolation_uncertainty
    loss = F.binary_cross_entropy_with_logits(
        logits, soft_target, reduction='none'
    )
    return (loss * weights).sum() / weights.sum().clamp_min(1.0)


def focal_tversky_loss(logits, target, alpha=0.50, beta=0.50, gamma=0.75):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return ((1.0 - tversky).clamp_min(0.0) ** gamma)


def segmentation_loss(logits, soft_target, binary_target, hard_loss_weight):
    dice = soft_dice_loss(logits, soft_target)
    boundary_bce = boundary_weighted_bce(logits, soft_target, binary_target)
    hard_tversky_per_case = focal_tversky_loss(logits, binary_target)
    
    
    hard_loss_weight = hard_loss_weight.reshape(-1, 1)
    hard_tversky = (
        hard_tversky_per_case * hard_loss_weight
    ).sum() / hard_loss_weight.sum().clamp_min(1.0)
    total = 0.55 * dice + 0.30 * boundary_bce + 0.15 * hard_tversky
    return total, {
        'soft_dice': dice,
        'boundary_bce': boundary_bce,
        'hard_focal_tversky': hard_tversky,
    }


def territory_presence_target(binary_target, region_map):
    targets = []
    for label in range(1, NUM_TERRITORY_NODES + 1):
        targets.append(
            ((binary_target > 0) & (region_map == label)).any().astype(np.float32)
        )
    return np.asarray(targets, dtype=np.float32)


eligible_real_df = train_target_df[
    (train_target_df['binary_voxels'] > 0)
    & (train_target_df['binary_loss_weight'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_real_df = eligible_real_df[
    eligible_real_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
candidate_midpoint = len(eligible_real_df) // 2
candidate_order = sorted(
    range(len(eligible_real_df)),
    key=lambda index: (abs(index - candidate_midpoint), index),
)
real_subject = None
for candidate_index in candidate_order:
    candidate_subject = str(
        eligible_real_df.iloc[candidate_index]['subject_id']
    )
    candidate_row = manifest_by_subject[candidate_subject]
    with np.load(
        Path(str(candidate_row['vascufate_v2_npz_path'])),
        allow_pickle=False,
    ) as candidate_data:
        candidate_target = np.asarray(
            candidate_data['target_binary'], dtype=np.uint8
        )
    candidate_region, _ = load_region_map(
        candidate_row, candidate_target.shape
    )
    if territory_presence_target(
        candidate_target, candidate_region
    ).sum() > 0:
        real_subject = candidate_subject
        break
if real_subject is None:
    raise RuntimeError(
        'No atlas-enabled training lesion overlaps a valid territory.'
    )
real_row = manifest_by_subject[real_subject]
real_graph = load_graph_inputs(real_row)
if file_sha256(Path(str(real_row['vascufate_v2_npz_path']))) != str(
    real_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{real_subject}: real-loss NPZ hash changed.')
with np.load(
    Path(str(real_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    real_image = np.asarray(data['image'], dtype=np.float32)
    real_binary = np.asarray(data['target_binary'], dtype=np.float32)
    real_soft = np.asarray(data['target_soft'], dtype=np.float32)
    real_crop_mask = np.asarray(data['crop_mask']) > 0
    real_binary_weight = float(np.asarray(data['binary_loss_weight']).item())
real_region, real_atlas_enabled = load_region_map(
    real_row, real_binary.shape
)
real_territory_target = territory_presence_target(real_binary, real_region)
real_center, real_center_source = deterministic_patch_center(
    real_binary, real_crop_mask, SEED + 9001, positive_probability=1.0
)
real_slices = patch_slices_from_center(
    real_binary.shape, real_center, PATCH_SIZE
)
real_image_patch = extract_patch(
    real_image, real_slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
)
real_binary_patch = extract_patch(
    real_binary, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_soft_patch = extract_patch(
    real_soft, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_region_patch = extract_patch(
    real_region, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)

real_batch = {
    'image': torch.from_numpy(real_image_patch[None]).to(device),
    'region_map': torch.from_numpy(real_region_patch[None]).to(device),
    'territory_features': torch.from_numpy(
        real_graph['territory_features'][None]
    ).to(device),
    'territory_feature_valid': torch.from_numpy(
        real_graph['territory_feature_valid'][None]
    ).to(device),
    'territory_valid': torch.from_numpy(
        real_graph['territory_valid'][None]
    ).to(device),
    'territory_reliability': torch.from_numpy(
        real_graph['territory_reliability'][None]
    ).to(device),
    'vessel_features': torch.from_numpy(
        real_graph['vessel_features'][None]
    ).to(device),
    'vessel_feature_valid': torch.from_numpy(
        real_graph['vessel_feature_valid'][None]
    ).to(device),
    'vessel_valid': torch.from_numpy(real_graph['vessel_valid'][None]).to(device),
    'case_features': torch.from_numpy(real_graph['case_features'][None]).to(device),
    'atlas_enabled': torch.as_tensor(
        [real_atlas_enabled], dtype=torch.bool, device=device
    ),
}
real_soft_tensor = torch.from_numpy(real_soft_patch[None, None]).to(device)
real_binary_tensor = torch.from_numpy(real_binary_patch[None, None]).to(device)
real_hard_weight = torch.as_tensor(
    [real_binary_weight], dtype=torch.float32, device=device
)
real_territory_target_tensor = torch.from_numpy(
    real_territory_target[None]
).to(device)


set_determinism(SEED + 1)
model = VascuFateNet(
    schema=graph_schema,
    image_channels=real_image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
amp_scaler = torch.amp.GradScaler(
    device.type, enabled=device.type == 'cuda'
)
before_step = model.segmentation_head.weight.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
    enabled=device.type == 'cuda',
):
    real_output = model(**real_batch, force_atlas_dropout=False)
    main_loss, main_parts = segmentation_loss(
        real_output['logits'],
        real_soft_tensor,
        real_binary_tensor,
        real_hard_weight,
    )
    half_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        real_output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        real_hard_weight,
    )
    quarter_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        real_output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        real_hard_weight,
    )
    node_mask = (
        real_batch['territory_valid'].bool()
        & real_output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        real_output['territory_node_logits'],
        real_territory_target_tensor,
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = real_output['territory_node_logits'].sum() * 0.0
    total_real_loss = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )

amp_scaler.scale(total_real_loss).backward()
amp_scaler.unscale_(optimizer)
real_gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0).item()
)
amp_scaler.step(optimizer)
amp_scaler.update()
optimizer.zero_grad(set_to_none=True)
parameter_update = float(
    (model.segmentation_head.weight.detach() - before_step).abs().max().item()
)

loss_values = {
    'subject_id': real_subject,
    'patch_center_source': real_center_source,
    'patch_target_voxels': int(real_binary_patch.sum()),
    'main_loss': float(main_loss.detach().cpu()),
    'soft_dice_loss': float(main_parts['soft_dice'].detach().cpu()),
    'boundary_bce_loss': float(main_parts['boundary_bce'].detach().cpu()),
    'hard_focal_tversky_loss': float(
        main_parts['hard_focal_tversky'].detach().cpu()
    ),
    'half_resolution_loss': float(half_loss.detach().cpu()),
    'quarter_resolution_loss': float(quarter_loss.detach().cpu()),
    'territory_node_loss': float(node_loss.detach().cpu()),
    'total_loss': float(total_real_loss.detach().cpu()),
    'gradient_norm_before_clipping': real_gradient_norm,
    'segmentation_head_maximum_update': parameter_update,
}
atomic_write_json(loss_values, LOSS_AUDIT_PATH)
loss_checks = {
    'real_patch_contains_lesion': int(real_binary_patch.sum()) > 0,
    'all_real_loss_components_finite': all(
        math.isfinite(float(value))
        for key, value in loss_values.items()
        if key.endswith('_loss') or key == 'total_loss'
    ),
    'real_gradient_norm_positive_and_finite': math.isfinite(real_gradient_norm)
    and real_gradient_norm > 0,
    'optimizer_step_changed_parameters': parameter_update > 0,
    'territory_target_has_30_nodes': real_territory_target.shape == (30,),
    'territory_target_contains_positive_node': real_territory_target.sum() > 0,
    'parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
}
if not all(loss_checks.values()):
    failed = [name for name, passed in loss_checks.items() if not passed]
    raise RuntimeError('Real-loss optimizer preflight failed: ' + ', '.join(failed))

print('\nReal-loss disposable optimizer-step audit:')
display(pd.DataFrame([
    {'metric': name, 'value': value} for name, value in loss_values.items()
]))
print('\nParameter count after model reinitialization:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': parameter_count(model)},
    {'component': 'trainable', 'parameters': parameter_count(model, True)},
]))






def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


@torch.no_grad()
def sliding_window_probability(model, row, image, region_map, graph_inputs):
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, PATCH_SIZE)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(PATCH_SIZE)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, PATCH_SIZE)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    model.eval()
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index : location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(
                np.repeat(graph_inputs['territory_features'][None], count, axis=0)
            ).to(device),
            'territory_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'territory_valid': torch.from_numpy(
                np.repeat(graph_inputs['territory_valid'][None], count, axis=0)
            ).to(device),
            'territory_reliability': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_reliability'][None], count, axis=0
                )
            ).to(device),
            'vessel_features': torch.from_numpy(
                np.repeat(graph_inputs['vessel_features'][None], count, axis=0)
            ).to(device),
            'vessel_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['vessel_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'vessel_valid': torch.from_numpy(
                np.repeat(graph_inputs['vessel_valid'][None], count, axis=0)
            ).to(device),
            'case_features': torch.from_numpy(
                np.repeat(graph_inputs['case_features'][None], count, axis=0)
            ).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **inference_batch, force_atlas_dropout=False
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    return probability[original_slices]


eligible_validation_df = target_audit_df[
    (target_audit_df['dev_split'] == 'val')
    & (target_audit_df['binary_voxels'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_validation_df = eligible_validation_df[
    eligible_validation_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
validation_subject = str(
    eligible_validation_df.iloc[len(eligible_validation_df) // 2]['subject_id']
)
validation_row = manifest_by_subject[validation_subject]
validation_graph = load_graph_inputs(validation_row)
if file_sha256(Path(str(validation_row['vascufate_v2_npz_path']))) != str(
    validation_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{validation_subject}: validation NPZ hash changed.')
with np.load(
    Path(str(validation_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    validation_image = np.asarray(data['image'], dtype=np.float32)
    validation_target = np.asarray(data['target_binary'], dtype=np.uint8)
validation_region, _ = load_region_map(
    validation_row, validation_target.shape
)
validation_probability = sliding_window_probability(
    model,
    validation_row,
    validation_image,
    validation_region,
    validation_graph,
)
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    prediction = validation_probability >= threshold
    intersection = int((prediction & (validation_target > 0)).sum())
    predicted = int(prediction.sum())
    truth = int(validation_target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else (2.0 * intersection) / (predicted + truth)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    precision = 1.0 if predicted == 0 and truth == 0 else (
        intersection / max(predicted, 1)
    )
    threshold_rows.append({
        'subject_id': validation_subject,
        'threshold': threshold,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    })
validation_metric_df = pd.DataFrame(threshold_rows)
atomic_write_csv(validation_metric_df, VALIDATION_AUDIT_PATH)
validation_checks = {
    'validation_probability_shape_matches': validation_probability.shape
    == validation_target.shape,
    'validation_probability_finite': bool(
        np.isfinite(validation_probability).all()
    ),
    'validation_probability_in_unit_interval': bool(
        validation_probability.min() >= 0
        and validation_probability.max() <= 1
    ),
    'five_thresholds_evaluated': len(validation_metric_df) == 5,
    'validation_metrics_finite': bool(
        np.isfinite(
            validation_metric_df[['dice', 'precision', 'recall']].to_numpy()
        ).all()
    ),
    'test_targets_still_not_opened': target_arrays_read_splits == {
        'train',
        'val',
    },
}
if not all(validation_checks.values()):
    failed = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Sliding-window validation preflight failed: ' + ', '.join(failed)
    )

print('\nDisposable full-volume validation smoke metrics:')
display(validation_metric_df)






training_policy = {
    'patch_size': list(PATCH_SIZE),
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'patches_per_subject_per_epoch': 2,
    'sampling_schedule': 'one_lesion_centered_plus_one_brain_context',
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'early_stopping_patience_validations': 10,
    'optimizer': 'AdamW',
    'learning_rate': 1e-4,
    'weight_decay': 5e-4,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'positive_center_probability': 1.0,
    'context_center_probability': 1.0,
    'intensity_gain_range': [0.90, 1.10],
    'intensity_offset_range': [-0.03, 0.03],
    'noise_probability': 0.65,
    'maximum_noise_standard_deviation': 0.02,
    'gamma_probability': 0.50,
    'gamma_range': [0.85, 1.15],
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'main_loss_weights': {
        'soft_dice': 0.55,
        'boundary_weighted_bce': 0.30,
        'hard_focal_tversky': 0.15,
    },
    'focal_tversky_alpha_false_positive': 0.50,
    'focal_tversky_beta_false_negative': 0.50,
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.05,
    'territory_positive_weight': 4.0,
    'validation_inference': '50_percent_overlap_gaussian_sliding_window',
    'threshold_candidates': [
        round(value, 2) for value in np.arange(0.10, 0.61, 0.05)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_training_checks = {
    **frozen_architecture_checks,
    **target_audit_checks,
    **patch_checks,
    **loss_checks,
    **validation_checks,
    'frozen_scaler_hash_matches': scaler_sha256 == EXPECTED_SCALER_SHA256,
    'target_free_model_recheck_passed': preflight_passed,
    'total_parameters_match': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameters_match': parameter_count(model, True)
    == EXPECTED_TOTAL_PARAMETERS,
    'test_split_remains_sealed': 'test' not in target_arrays_read_splits,
}
training_preflight_passed = bool(all(all_training_checks.values()))
atomic_write_csv(
    pd.DataFrame([
        {'check': name, 'passed': passed}
        for name, passed in all_training_checks.items()
    ]),
    PREFLIGHT_AUDIT_PATH,
)
final_gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'training_preflight_passed': training_preflight_passed,
    'full_training_authorized': training_preflight_passed,
    'checks': all_training_checks,
    'training_policy': training_policy,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_sha256': architecture_sha256,
    'total_parameters': parameter_count(model),
    'trainable_parameters': parameter_count(model, True),
    'real_loss_subject': real_subject,
    'validation_smoke_subject': validation_subject,
    'target_arrays_read_splits': sorted(target_arrays_read_splits),
    'test_targets_read': False,
    'study_training_epochs_completed': 0,
}
atomic_write_json(final_gate, PREFLIGHT_GATE_PATH)

print('\nFinal target-aware training preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_training_checks.items()
]))
if not training_preflight_passed:
    failed = [
        name for name, passed in all_training_checks.items() if not passed
    ]
    raise RuntimeError(
        'TRIAL8 TARGET-AWARE TRAINING PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT PASSED')
print('=' * 80)
print('Train targets audited:              102')
print('Validation targets audited:         23')
print('Test targets read:                  False')
print(f'Real-loss subject:                  {real_subject}')
print(f'Validation smoke subject:           {validation_subject}')
print(f'Total parameters:                   {parameter_count(model):,}')
print(f'Trainable parameters:               {parameter_count(model, True):,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Frozen scaler SHA-256:              {scaler_sha256}')
print(f'Training preflight gate:            {PREFLIGHT_GATE_PATH}')
print('Study training epochs completed:    0')
print('Full Trial8 training authorized:    True')
print('\nTarget-aware gate passed; starting checkpoint diagnostic below.')








del model, optimizer, amp_scaler, real_output, real_batch
del real_soft_tensor, real_binary_tensor, real_territory_target_tensor
del normal_output, dropped_output, gradient_output, batch
del smoke_loss, total_real_loss, main_loss, half_loss, quarter_loss
del node_loss, node_bce, before_step
del validation_probability, validation_image, validation_target
del validation_region, validation_graph
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DIAGNOSTIC_VERSION = 'v1_dense_threshold_v1_v2_probability_ensemble'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
V1_BEST_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / 'v1_ragatv2_film_boundary_region'
    / 'best_checkpoint.pt'
)
V2_BEST_CHECKPOINT_PATH = (
    CHECKPOINT_ROOT
    / 'v2_regularized_balanced_epochwise'
    / 'best_checkpoint.pt'
)
DIAGNOSTIC_ROOT = CHECKPOINT_ROOT / DIAGNOSTIC_VERSION
CASE_METRICS_PATH = DIAGNOSTIC_ROOT / 'validation_case_metrics.csv'
CONFIGURATION_SUMMARY_PATH = (
    DIAGNOSTIC_ROOT / 'validation_configuration_summary.csv'
)
SELECTED_CONFIGURATIONS_PATH = (
    DIAGNOSTIC_ROOT / 'selected_configurations.csv'
)
LESION_BIN_SUMMARY_PATH = (
    DIAGNOSTIC_ROOT / 'selected_lesion_bin_summary.csv'
)
PAIRED_BOOTSTRAP_PATH = (
    DIAGNOSTIC_ROOT / 'paired_bootstrap_differences.csv'
)
DIAGNOSTIC_GATE_PATH = DIAGNOSTIC_ROOT / 'diagnostic_gate.json'

for path in [
    V1_BEST_CHECKPOINT_PATH,
    V2_BEST_CHECKPOINT_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            'Required best checkpoint is missing: ' + str(path)
        )
assert_project_path_is_safe(DIAGNOSTIC_ROOT)
DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)



DENSE_THRESHOLDS = np.round(
    np.arange(0.05, 0.9501, 0.025), 3
).astype(np.float64)
V2_ENSEMBLE_WEIGHTS = np.asarray(
    [0.00, 0.25, 0.50, 0.75, 1.00], dtype=np.float64
)
V2_REPORTED_THRESHOLD = 0.60
BOOTSTRAP_REPLICATES = 10_000


def build_study_model():
    study_model = VascuFateNet(
        schema=graph_schema,
        image_channels=15,
        territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
        vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
        case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
        graph_hidden=GRAPH_HIDDEN,
        graph_heads=GRAPH_HEADS,
        graph_layers=GRAPH_LAYERS,
        relation_bases=RELATION_BASES,
        graph_dropout=GRAPH_DROPOUT,
        atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
        film_bound=FILM_BOUND,
    ).to(device)
    if parameter_count(study_model) != EXPECTED_TOTAL_PARAMETERS:
        raise RuntimeError('Diagnostic model parameter count changed.')
    return study_model


def load_read_only_checkpoint(path, label):
    checkpoint = torch.load(
        path, map_location='cpu', weights_only=False
    )
    checks = {
        'model_state_dict_present': 'model_state_dict' in checkpoint,
        'manifest_hash_matches': checkpoint.get(
            'verified_manifest_sha256'
        ) == EXPECTED_VERIFIED_MANIFEST_SHA256,
        'scaler_hash_matches': checkpoint.get(
            'scaler_sha256'
        ) == EXPECTED_SCALER_SHA256,
        'graph_schema_hash_matches': checkpoint.get(
            'graph_schema_sha256'
        ) == EXPECTED_SAFE_SCHEMA_SHA256,
        'parameter_count_matches': checkpoint.get(
            'total_parameters'
        ) == EXPECTED_TOTAL_PARAMETERS,
        'test_targets_were_not_read': checkpoint.get(
            'test_targets_read'
        ) is False,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'{label} checkpoint integrity failed: ' + ', '.join(failed)
        )
    study_model = build_study_model()
    study_model.load_state_dict(
        checkpoint['model_state_dict'], strict=True
    )
    study_model.eval()
    metadata = {
        'checkpoint_label': label,
        'checkpoint_path': str(path),
        'checkpoint_sha256': file_sha256(path),
        'training_version': str(checkpoint.get('training_version', '')),
        'best_epoch': int(checkpoint.get('best_epoch', checkpoint['epoch'])),
        'reported_best_mean_dice': float(
            checkpoint.get('best_mean_dice', np.nan)
        ),
        'reported_best_threshold': float(
            checkpoint.get('best_threshold', np.nan)
        ),
        'total_parameters': parameter_count(study_model),
        'trainable_parameters': parameter_count(study_model, True),
    }
    return study_model, metadata


def threshold_counts(probability, target, thresholds):
    probability = np.asarray(probability, dtype=np.float32).reshape(-1)
    target = np.asarray(target, dtype=bool).reshape(-1)
    if probability.shape != target.shape:
        raise RuntimeError('Probability and target shapes differ.')
    if not np.isfinite(probability).all():
        raise RuntimeError('Non-finite validation probabilities found.')
    if float(probability.min()) < 0.0 or float(probability.max()) > 1.0:
        raise RuntimeError('Validation probabilities are outside [0, 1].')
    
    
    edges = np.concatenate((
        np.asarray([-np.inf], dtype=np.float32),
        np.asarray(thresholds, dtype=np.float32),
        np.asarray([np.inf], dtype=np.float32),
    ))
    positive_histogram = np.histogram(
        probability[target], bins=edges
    )[0].astype(np.int64)
    negative_histogram = np.histogram(
        probability[~target], bins=edges
    )[0].astype(np.int64)
    true_positive = np.cumsum(positive_histogram[::-1])[::-1][1:]
    false_positive = np.cumsum(negative_histogram[::-1])[::-1][1:]
    truth = int(target.sum())
    predicted = true_positive + false_positive
    false_negative = truth - true_positive
    denominator = predicted + truth
    dice = np.divide(
        2.0 * true_positive,
        denominator,
        out=np.ones_like(denominator, dtype=np.float64),
        where=denominator > 0,
    )
    precision = np.divide(
        true_positive,
        predicted,
        out=np.where(truth == 0, 1.0, 0.0).astype(np.float64)
        * np.ones_like(predicted, dtype=np.float64),
        where=predicted > 0,
    )
    recall = (
        np.ones_like(true_positive, dtype=np.float64)
        if truth == 0
        else true_positive.astype(np.float64) / truth
    )
    return {
        'intersection': true_positive,
        'false_positive_voxels': false_positive,
        'false_negative_voxels': false_negative,
        'predicted_voxels': predicted,
        'target_voxels': np.full_like(predicted, truth),
        'dice': dice,
        'precision': precision,
        'recall': recall,
    }


def summarize_configurations(case_frame):
    grouped = (
        case_frame.groupby(['v2_weight', 'threshold'], as_index=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_subject_dice=('dice', 'mean'),
            median_subject_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_error_ml=(
                'absolute_volume_error_ml', 'mean'
            ),
            total_intersection=('intersection', 'sum'),
            total_predicted_voxels=('predicted_voxels', 'sum'),
            total_target_voxels=('target_voxels', 'sum'),
        )
    )
    grouped['global_dice'] = (
        2.0 * grouped['total_intersection']
        / np.maximum(
            grouped['total_predicted_voxels']
            + grouped['total_target_voxels'],
            1,
        )
    )
    grouped['v1_weight'] = 1.0 - grouped['v2_weight']
    grouped['model'] = grouped['v2_weight'].map(
        lambda value: (
            'v1'
            if np.isclose(value, 0.0)
            else 'v2'
            if np.isclose(value, 1.0)
            else f'v1_v2_ensemble_v2_weight_{value:.2f}'
        )
    )
    return grouped.sort_values(
        [
            'mean_subject_dice',
            'median_subject_dice',
            'mean_recall',
            'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)


def select_best(summary, weights=None):
    candidates = summary
    if weights is not None:
        candidates = candidates[
            candidates['v2_weight'].map(
                lambda value: any(np.isclose(value, item) for item in weights)
            )
        ]
    if candidates.empty:
        raise RuntimeError('No diagnostic configuration can be selected.')
    return candidates.iloc[0].to_dict()


def configuration_cases(case_frame, configuration):
    return case_frame[
        np.isclose(case_frame['v2_weight'], configuration['v2_weight'])
        & np.isclose(case_frame['threshold'], configuration['threshold'])
    ].sort_values('subject_id').reset_index(drop=True)


def paired_bootstrap_difference(reference, candidate, label, seed):
    merged = reference[
        ['subject_id', 'dice']
    ].merge(
        candidate[['subject_id', 'dice']],
        on='subject_id',
        suffixes=('_reference', '_candidate'),
        validate='one_to_one',
    )
    differences = (
        merged['dice_candidate'] - merged['dice_reference']
    ).to_numpy(dtype=np.float64)
    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(
        0,
        len(differences),
        size=(BOOTSTRAP_REPLICATES, len(differences)),
    )
    bootstrap_means = differences[sample_indices].mean(axis=1)
    return {
        'comparison': label,
        'subjects': len(differences),
        'mean_dice_difference': float(differences.mean()),
        'median_dice_difference': float(np.median(differences)),
        'subjects_improved': int((differences > 0).sum()),
        'subjects_tied': int(np.isclose(differences, 0.0).sum()),
        'subjects_worsened': int((differences < 0).sum()),
        'descriptive_bootstrap_ci95_low': float(
            np.quantile(bootstrap_means, 0.025)
        ),
        'descriptive_bootstrap_ci95_high': float(
            np.quantile(bootstrap_means, 0.975)
        ),
        'note': (
            'Descriptive only: configurations were selected on this same '
            'validation split.'
        ),
    }


v1_model, v1_metadata = load_read_only_checkpoint(
    V1_BEST_CHECKPOINT_PATH, 'v1_baseline_best'
)
v2_model, v2_metadata = load_read_only_checkpoint(
    V2_BEST_CHECKPOINT_PATH, 'v2_regularized_best'
)

checkpoint_metadata_df = pd.DataFrame([v1_metadata, v2_metadata])
print('\nRead-only checkpoint metadata:')
display(checkpoint_metadata_df)

validation_rows = (
    manifest_df[manifest_df['dev_split'] == 'val']
    .sort_values('subject_id')
    .to_dict('records')
)
if len(validation_rows) != EXPECTED_SPLITS['val']:
    raise RuntimeError('Expected exactly 23 validation subjects.')

def run_checkpoint_inference(rows):
    case_rows = []
    for case_index, row in enumerate(rows, start=1):
        subject_id = str(row['subject_id'])
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
            raise RuntimeError(f'{subject_id}: validation NPZ hash changed.')
        with np.load(npz_path, allow_pickle=False) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target = np.asarray(data['target_binary'], dtype=np.uint8)
            spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        region_map, _ = load_region_map(row, target.shape)
        graph_inputs = load_graph_inputs(row)
        v1_probability = sliding_window_probability(
            v1_model, row, image, region_map, graph_inputs
        )
        v2_probability = sliding_window_probability(
            v2_model, row, image, region_map, graph_inputs
        )
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        lesion_bin = str(row.get('lesion_volume_bin', 'unknown'))
        for v2_weight in V2_ENSEMBLE_WEIGHTS:
            probability = (
                (1.0 - float(v2_weight)) * v1_probability
                + float(v2_weight) * v2_probability
            )
            metric_arrays = threshold_counts(
                probability, target > 0, DENSE_THRESHOLDS
            )
            for threshold_index, threshold in enumerate(DENSE_THRESHOLDS):
                predicted_voxels = int(
                    metric_arrays['predicted_voxels'][threshold_index]
                )
                target_voxels = int(
                    metric_arrays['target_voxels'][threshold_index]
                )
                case_rows.append({
                    'subject_id': subject_id,
                    'lesion_volume_bin': lesion_bin,
                    'v2_weight': float(v2_weight),
                    'v1_weight': float(1.0 - v2_weight),
                    'threshold': float(threshold),
                    'dice': float(metric_arrays['dice'][threshold_index]),
                    'precision': float(
                        metric_arrays['precision'][threshold_index]
                    ),
                    'recall': float(metric_arrays['recall'][threshold_index]),
                    'intersection': int(
                        metric_arrays['intersection'][threshold_index]
                    ),
                    'false_positive_voxels': int(
                        metric_arrays[
                            'false_positive_voxels'
                        ][threshold_index]
                    ),
                    'false_negative_voxels': int(
                        metric_arrays[
                            'false_negative_voxels'
                        ][threshold_index]
                    ),
                    'predicted_voxels': predicted_voxels,
                    'target_voxels': target_voxels,
                    'predicted_volume_ml': (
                        predicted_voxels * voxel_volume_ml
                    ),
                    'target_volume_ml': target_voxels * voxel_volume_ml,
                    'absolute_volume_error_ml': (
                        abs(predicted_voxels - target_voxels)
                        * voxel_volume_ml
                    ),
                    'target_arrays_read_from_split': 'val',
                })
        del image, target, region_map, graph_inputs
        del v1_probability, v2_probability
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if (
            case_index == 1
            or case_index % 5 == 0
            or case_index == len(rows)
        ):
            print(
                f'Evaluated {case_index}/{len(rows)}: {subject_id}'
            )
    return pd.DataFrame(case_rows)


def reusable_case_metrics(frame, prior_gate, rows):
    required_columns = {
        'subject_id',
        'lesion_volume_bin',
        'v2_weight',
        'v1_weight',
        'threshold',
        'dice',
        'precision',
        'recall',
        'intersection',
        'predicted_voxels',
        'target_voxels',
        'absolute_volume_error_ml',
        'target_arrays_read_from_split',
    }
    if not required_columns.issubset(frame.columns):
        return False
    expected_subject_ids = {str(row['subject_id']) for row in rows}
    observed_subject_ids = set(frame['subject_id'].astype(str))
    observed_weights = np.sort(frame['v2_weight'].unique())
    observed_thresholds = np.sort(frame['threshold'].unique())
    expected_rows = (
        len(rows)
        * len(V2_ENSEMBLE_WEIGHTS)
        * len(DENSE_THRESHOLDS)
    )
    provenance_matches = (
        prior_gate.get('diagnostic_version') == DIAGNOSTIC_VERSION
        and prior_gate.get('diagnostic_passed') is True
        and prior_gate.get('test_targets_read') is False
        and prior_gate.get('training_performed') is False
        and prior_gate.get('v1_checkpoint', {}).get('checkpoint_sha256')
        == v1_metadata['checkpoint_sha256']
        and prior_gate.get('v2_checkpoint', {}).get('checkpoint_sha256')
        == v2_metadata['checkpoint_sha256']
    )
    return bool(
        provenance_matches
        and len(frame) == expected_rows
        and observed_subject_ids == expected_subject_ids
        and len(observed_weights) == len(V2_ENSEMBLE_WEIGHTS)
        and np.allclose(observed_weights, V2_ENSEMBLE_WEIGHTS)
        and len(observed_thresholds) == len(DENSE_THRESHOLDS)
        and np.allclose(observed_thresholds, DENSE_THRESHOLDS)
        and set(frame['target_arrays_read_from_split'].astype(str))
        == {'val'}
        and not frame.duplicated(
            ['subject_id', 'v2_weight', 'threshold']
        ).any()
        and np.allclose(
            frame['v1_weight'].to_numpy(dtype=np.float64),
            1.0 - frame['v2_weight'].to_numpy(dtype=np.float64),
        )
        and np.isfinite(
            frame[
                [
                    'dice',
                    'precision',
                    'recall',
                    'absolute_volume_error_ml',
                ]
            ].to_numpy(dtype=np.float64)
        ).all()
    )


cached_case_metrics_reused = False
case_metrics_df = None
if CASE_METRICS_PATH.exists() and DIAGNOSTIC_GATE_PATH.exists():
    try:
        cached_frame = pd.read_csv(CASE_METRICS_PATH)
        prior_gate = json.loads(
            DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8')
        )
        if reusable_case_metrics(
            cached_frame, prior_gate, validation_rows
        ):
            case_metrics_df = cached_frame
            cached_case_metrics_reused = True
            print(
                'Reusing hash-matched completed validation case metrics; '
                'checkpoint inference will not be repeated.'
            )
    except (OSError, ValueError, KeyError, TypeError, json.JSONDecodeError):
        case_metrics_df = None

if case_metrics_df is None:
    case_metrics_df = run_checkpoint_inference(validation_rows)

configuration_summary_df = summarize_configurations(case_metrics_df)

v1_best = select_best(configuration_summary_df, weights=[0.0])
v2_dense_best = select_best(configuration_summary_df, weights=[1.0])
ensemble_best = select_best(
    configuration_summary_df, weights=[0.25, 0.50, 0.75]
)
overall_best = select_best(configuration_summary_df)
v2_reported_row = configuration_summary_df[
    np.isclose(configuration_summary_df['v2_weight'], 1.0)
    & np.isclose(
        configuration_summary_df['threshold'], V2_REPORTED_THRESHOLD
    )
]
if len(v2_reported_row) != 1:
    raise RuntimeError('Could not recover the v2 threshold-0.60 baseline.')
v2_reported = v2_reported_row.iloc[0].to_dict()

selected_rows = []
for selection_name, record in [
    ('v2_reported_threshold', v2_reported),
    ('v1_dense_best', v1_best),
    ('v2_dense_best', v2_dense_best),
    ('ensemble_dense_best', ensemble_best),
    ('overall_dense_best', overall_best),
]:
    selected_rows.append({'selection': selection_name, **record})
selected_configurations_df = pd.DataFrame(selected_rows)

lesion_bin_rows = []
selected_case_frames = {}
for record in selected_rows:
    selection = str(record['selection'])
    selected_cases = configuration_cases(case_metrics_df, record)
    selected_case_frames[selection] = selected_cases
    grouped = (
        selected_cases.groupby('lesion_volume_bin', dropna=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_dice=('dice', 'mean'),
            median_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
        )
        .reset_index()
    )
    grouped.insert(0, 'selection', selection)
    lesion_bin_rows.extend(grouped.to_dict('records'))
lesion_bin_summary_df = pd.DataFrame(lesion_bin_rows)

bootstrap_rows = [
    paired_bootstrap_difference(
        selected_case_frames['v2_reported_threshold'],
        selected_case_frames['v2_dense_best'],
        'v2_dense_minus_v2_reported',
        SEED + 101,
    ),
    paired_bootstrap_difference(
        selected_case_frames['v2_reported_threshold'],
        selected_case_frames['ensemble_dense_best'],
        'ensemble_dense_minus_v2_reported',
        SEED + 102,
    ),
    paired_bootstrap_difference(
        selected_case_frames['v2_dense_best'],
        selected_case_frames['ensemble_dense_best'],
        'ensemble_dense_minus_v2_dense',
        SEED + 103,
    ),
]
paired_bootstrap_df = pd.DataFrame(bootstrap_rows)

diagnostic_checks = {
    'exactly_23_validation_subjects': (
        case_metrics_df['subject_id'].nunique() == 23
    ),
    'only_validation_targets_read': set(
        case_metrics_df['target_arrays_read_from_split'].unique()
    ) == {'val'},
    'test_targets_not_read': 'test' not in target_arrays_read_splits,
    'five_checkpoint_weight_settings_evaluated': (
        case_metrics_df['v2_weight'].nunique() == 5
    ),
    'thirty_seven_dense_thresholds_evaluated': (
        case_metrics_df['threshold'].nunique() == 37
    ),
    'all_metrics_finite': bool(np.isfinite(
        case_metrics_df[
            ['dice', 'precision', 'recall', 'absolute_volume_error_ml']
        ].to_numpy()
    ).all()),
    'all_probabilistic_metrics_bounded': bool(
        case_metrics_df[['dice', 'precision', 'recall']]
        .ge(0.0).all().all()
        and case_metrics_df[['dice', 'precision', 'recall']]
        .le(1.0).all().all()
    ),
    'source_checkpoints_unchanged': (
        file_sha256(V1_BEST_CHECKPOINT_PATH)
        == v1_metadata['checkpoint_sha256']
        and file_sha256(V2_BEST_CHECKPOINT_PATH)
        == v2_metadata['checkpoint_sha256']
    ),
    'architecture_parameter_count_unchanged': (
        parameter_count(v1_model) == EXPECTED_TOTAL_PARAMETERS
        and parameter_count(v2_model) == EXPECTED_TOTAL_PARAMETERS
    ),
    'scientific_core_unchanged': True,
    'no_training_performed': True,
}
diagnostic_passed = bool(all(diagnostic_checks.values()))

atomic_write_csv(case_metrics_df, CASE_METRICS_PATH)
atomic_write_csv(
    configuration_summary_df, CONFIGURATION_SUMMARY_PATH
)
atomic_write_csv(
    selected_configurations_df, SELECTED_CONFIGURATIONS_PATH
)
atomic_write_csv(lesion_bin_summary_df, LESION_BIN_SUMMARY_PATH)
atomic_write_csv(paired_bootstrap_df, PAIRED_BOOTSTRAP_PATH)

best_threshold_at_boundary = bool(
    np.isclose(overall_best['threshold'], DENSE_THRESHOLDS.min())
    or np.isclose(overall_best['threshold'], DENSE_THRESHOLDS.max())
)
atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': DIAGNOSTIC_VERSION,
        'diagnostic_passed': diagnostic_passed,
        'checks': diagnostic_checks,
        'validation_subjects': 23,
        'test_targets_read': False,
        'training_performed': False,
        'cached_case_metrics_reused': cached_case_metrics_reused,
        'study_trainable_parameters': 0,
        'model_parameters_per_checkpoint': EXPECTED_TOTAL_PARAMETERS,
        'scientific_core': (
            '15-channel residual 3D U-Net plus heterogeneous relation-aware '
            'GATv2 with bounded zero-init multiscale FiLM'
        ),
        'scientific_core_changed': False,
        'v1_checkpoint': v1_metadata,
        'v2_checkpoint': v2_metadata,
        'dense_thresholds': DENSE_THRESHOLDS.tolist(),
        'v2_ensemble_weights': V2_ENSEMBLE_WEIGHTS.tolist(),
        'selected_configurations': [
            to_builtin(record) for record in selected_rows
        ],
        'overall_best_threshold_at_search_boundary': (
            best_threshold_at_boundary
        ),
        'v3_training_authorized': False,
        'next_action': (
            'Check calibration and ensemble evidence before freezing the '
            'v3 imbalance-aware training policy.'
        ),
    },
    DIAGNOSTIC_GATE_PATH,
)

print('\nDense-threshold and ensemble selections:')
display(selected_configurations_df[[
    'selection',
    'model',
    'v1_weight',
    'v2_weight',
    'threshold',
    'mean_subject_dice',
    'median_subject_dice',
    'global_dice',
    'mean_precision',
    'mean_recall',
    'mean_absolute_volume_error_ml',
]])

print('\nSelected performance by lesion-volume bin:')
display(lesion_bin_summary_df)

print('\nPaired validation differences:')
display(paired_bootstrap_df)

print('\nDiagnostic checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in diagnostic_checks.items()
]))
if not diagnostic_passed:
    failed = [
        name for name, passed in diagnostic_checks.items() if not passed
    ]
    raise RuntimeError(
        'TRIAL8 THRESHOLD/ENSEMBLE DIAGNOSTIC FAILED. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 DENSE-THRESHOLD AND ENSEMBLE DIAGNOSTIC PASSED')
print('=' * 80)
print('Validation subjects:                23')
print('Test targets read:                  False')
print('Training performed:                 False')
print(f'Cached case metrics reused:          {cached_case_metrics_reused}')
print('Study trainable parameters:         0 (diagnostic only)')
print(f'Model parameters per checkpoint:    {EXPECTED_TOTAL_PARAMETERS:,}')
print('Architecture/scientific core changed: False')
print(f'V1 dense best mean Dice:            {v1_best["mean_subject_dice"]:.4f}')
print(f'V2 reported mean Dice:              {v2_reported["mean_subject_dice"]:.4f}')
print(f'V2 dense best mean Dice:            {v2_dense_best["mean_subject_dice"]:.4f}')
print(f'Ensemble dense best mean Dice:      {ensemble_best["mean_subject_dice"]:.4f}')
print(f'Overall best model:                 {overall_best["model"]}')
print(f'Overall best threshold:             {overall_best["threshold"]:.3f}')
print(f'Overall best mean Dice:             {overall_best["mean_subject_dice"]:.4f}')
print(f'Overall threshold at boundary:      {best_threshold_at_boundary}')
print('V3 training authorized:             False')
print(f'Case metrics:                       {CASE_METRICS_PATH}')
print(f'Configuration summary:              {CONFIGURATION_SUMMARY_PATH}')
print(f'Diagnostic gate:                    {DIAGNOSTIC_GATE_PATH}')
print('\nNo source checkpoint was modified.')
print('Stop here and check the calibration and ensemble evidence before v3.')


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
V3_PREFLIGHT_VERSION = 'v3_topology_curriculum_clinical_windows'
CODE_REVISION = 'r1_train_only_topology_negative_audit'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_PARAMETERS = 15_052_628
EXPECTED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_VERIFICATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / VERIFICATION_VERSION
)
GRAPH_VERIFICATION_GATE_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_verification_gate.json'
)
GRAPH_SCHEMA_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)
MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / 'v1_train_only_scaler_ragatv2_film'
)
MODEL_PREFLIGHT_GATE_PATH = (
    MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'
)
SCALER_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
THRESHOLD_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT
    / 'checkpoints'
    / MODEL_VERSION
    / 'v1_dense_threshold_v1_v2_probability_ensemble'
)
THRESHOLD_DIAGNOSTIC_GATE_PATH = (
    THRESHOLD_DIAGNOSTIC_ROOT / 'diagnostic_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / V3_PREFLIGHT_VERSION
)
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_target_audit.csv'
SAMPLING_AUDIT_PATH = OUTPUT_ROOT / 'topology_sampling_audit.csv'
WINDOW_AUDIT_PATH = OUTPUT_ROOT / 'clinical_window_audit.csv'
POLICY_PATH = OUTPUT_ROOT / 'v3_training_policy.json'
GATE_PATH = OUTPUT_ROOT / 'v3_sampler_loss_preflight_gate.json'

SEED = 20260920
PATCH_SIZE = (112, 112, 48)
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
POSITIVE_SAMPLES = 102
TOPOLOGY_NEGATIVE_SAMPLES = 51
BRAIN_NEGATIVE_SAMPLES = 51
EPOCH_SAMPLES = (
    POSITIVE_SAMPLES
    + TOPOLOGY_NEGATIVE_SAMPLES
    + BRAIN_NEGATIVE_SAMPLES
)
MAX_CENTER_CANDIDATES = 512
CASE_WEIGHT_MIN = 0.75
CASE_WEIGHT_MAX = 2.50


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    return str(value).strip().lower() in {'true', '1', 'yes'}


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    return value


def atomic_write_json(record, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


required_paths = [
    VERIFIED_MANIFEST_PATH,
    GRAPH_VERIFICATION_GATE_PATH,
    GRAPH_SCHEMA_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
    SCALER_PATH,
    THRESHOLD_DIAGNOSTIC_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 v3 preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
scaler_sha256 = file_sha256(SCALER_PATH)
if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Verified manifest hash changed.')
if schema_sha256 != EXPECTED_SCHEMA_SHA256:
    raise RuntimeError('Verified graph schema hash changed.')
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Train-only scaler hash changed.')

graph_gate = json.loads(
    GRAPH_VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
diagnostic_gate = json.loads(
    THRESHOLD_DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8')
)

upstream_checks = {
    'graph_verification_passed': graph_gate.get(
        'verification_passed'
    ) is True,
    'graph_training_authorized': graph_gate.get(
        'model_training_authorized'
    ) is True,
    'model_preflight_passed': model_gate.get(
        'preflight_passed'
    ) is True,
    'model_training_authorized': model_gate.get(
        'model_training_authorized'
    ) is True,
    'model_parameter_count_unchanged': model_gate.get(
        'total_parameters'
    ) == EXPECTED_PARAMETERS,
    'threshold_diagnostic_passed': diagnostic_gate.get(
        'diagnostic_passed'
    ) is True,
    'threshold_diagnostic_target_safe': diagnostic_gate.get(
        'test_targets_read'
    ) is False,
    'threshold_diagnostic_architecture_unchanged': diagnostic_gate.get(
        'scientific_core_changed'
    ) is False,
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream v3 gates failed: ' + ', '.join(failed))

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'lesion_volume_bin',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(
        'Verified manifest is missing columns: ' + ', '.join(missing_columns)
    )
manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Manifest checks failed: ' + ', '.join(failed))

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}
territory_edge_index = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)

print('Trial8 v3 upstream gates passed.')
print(f'Verified subjects:                  {len(manifest_df)}')
print(f'Split counts:                       {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Frozen manifest SHA-256:            {manifest_sha256}')
print(f'Pickle-free schema SHA-256:         {schema_sha256}')
print('Target arrays authorized:           train only')
print('Test targets read:                  False')
print('Study trainable parameters:         0 (preflight only)')





train_df = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .reset_index(drop=True)
)
target_rows = []
for row in train_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'crop_mask',
            'target_spacing',
            'image',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing arrays: {missing}')
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask']) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        image_shape = tuple(int(value) for value in data['image'].shape)
    if target.shape != soft.shape or target.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if image_shape[0] != 15 or image_shape[1:] != target.shape:
        raise RuntimeError(f'{subject_id}: image shape is inconsistent.')
    if not np.isfinite(soft).all():
        raise RuntimeError(f'{subject_id}: soft target is non-finite.')
    target_voxels = int(target.sum())
    volume_ml = float(target_voxels * np.prod(spacing) / 1000.0)
    target_rows.append({
        'subject_id': subject_id,
        'lesion_volume_bin': str(row['lesion_volume_bin']),
        'target_voxels': target_voxels,
        'lesion_volume_ml': volume_ml,
        'target_inside_crop_fraction': (
            1.0
            if target_voxels == 0
            else float((target.astype(bool) & crop_mask).sum() / target_voxels)
        ),
        'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
        'npz_path': str(npz_path),
    })

target_audit_df = pd.DataFrame(target_rows)
nonempty_volumes = target_audit_df.loc[
    target_audit_df['lesion_volume_ml'] > 0,
    'lesion_volume_ml',
].to_numpy(dtype=np.float64)
median_nonempty_volume_ml = float(np.median(nonempty_volumes))


def lesion_case_weight(volume_ml):
    if volume_ml <= 0:
        return 1.0
    return float(np.clip(
        math.sqrt(median_nonempty_volume_ml / volume_ml),
        CASE_WEIGHT_MIN,
        CASE_WEIGHT_MAX,
    ))


target_audit_df['positive_case_loss_weight'] = target_audit_df[
    'lesion_volume_ml'
].map(lesion_case_weight)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)

target_by_subject = target_audit_df.set_index('subject_id').to_dict('index')
manifest_by_subject = {
    str(row['subject_id']): row for row in train_df.to_dict('records')
}






def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def integral_volume(binary):
    return np.pad(
        np.asarray(binary, dtype=np.int64),
        ((1, 0), (1, 0), (1, 0)),
        mode='constant',
    ).cumsum(0).cumsum(1).cumsum(2)


def box_sum(integral, slices):
    x0, x1 = slices[0].start, slices[0].stop
    y0, y1 = slices[1].start, slices[1].stop
    z0, z1 = slices[2].start, slices[2].stop
    return int(
        integral[x1, y1, z1]
        - integral[x0, y1, z1]
        - integral[x1, y0, z1]
        - integral[x1, y1, z0]
        + integral[x0, y0, z1]
        + integral[x0, y1, z0]
        + integral[x1, y0, z0]
        - integral[x0, y0, z0]
    )


def sampled_coordinates(mask, rng, maximum=MAX_CENTER_CANDIDATES):
    flat = np.flatnonzero(np.asarray(mask, dtype=bool))
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > maximum:
        flat = rng.choice(flat, size=maximum, replace=False)
    coordinates = np.column_stack(
        np.unravel_index(flat, mask.shape)
    )
    return coordinates.astype(np.int64)


def minimum_overlap_center(target, candidate_mask, rng):
    coordinates = sampled_coordinates(candidate_mask, rng)
    if len(coordinates) == 0:
        return None, None
    integral = integral_volume(target > 0)
    counts = np.asarray([
        box_sum(
            integral,
            patch_slices_from_center(target.shape, center, PATCH_SIZE),
        )
        for center in coordinates
    ], dtype=np.int64)
    minimum = int(counts.min())
    best = np.flatnonzero(counts == minimum)
    selected = int(best[int(rng.integers(0, len(best)))])
    return coordinates[selected], minimum


def load_train_case(subject_id, include_image=False):
    row = manifest_by_subject[subject_id]
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
        image = (
            np.asarray(data['image'], dtype=np.float32)
            if include_image
            else None
        )
    region_map = np.zeros(target.shape, dtype=np.int16)
    if as_bool(row['trial8_atlas_enabled']):
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region_map = np.asanyarray(atlas_image.dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int16)
        if region_map.shape != target.shape:
            raise RuntimeError(f'{subject_id}: atlas shape mismatch.')
    return target, crop_mask, region_map, image


def topology_candidate_labels(target, region_map):
    lesion_labels = set(
        int(value)
        for value in np.unique(region_map[target > 0])
        if 1 <= int(value) <= 30
    )
    if not lesion_labels:
        return [], {}, lesion_labels
    candidates_by_priority = {2: [], 1: [], 3: []}
    relation_by_label = {}
    relation_priority = {2: 0, 1: 1, 3: 2}
    for edge_position in range(territory_edge_index.shape[1]):
        source = int(territory_edge_index[0, edge_position]) + 1
        destination = int(territory_edge_index[1, edge_position]) + 1
        relation = int(territory_edge_type[edge_position])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in candidates_by_priority
        ):
            candidates_by_priority[relation].append(destination)
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    ordered = []
    for relation in [2, 1, 3]:
        ordered.extend(sorted(set(candidates_by_priority[relation])))
    return list(dict.fromkeys(ordered)), relation_by_label, lesion_labels


RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def select_positive(subject_id, rng):
    target, crop_mask, region_map, _ = load_train_case(subject_id)
    positive = np.argwhere(target > 0)
    if len(positive) == 0:
        raise RuntimeError(f'{subject_id}: positive schedule selected empty target.')
    center = positive[int(rng.integers(0, len(positive)))].astype(np.int64)
    jitter = np.asarray([
        int(rng.integers(-12, 13)),
        int(rng.integers(-12, 13)),
        int(rng.integers(-4, 5)),
    ])
    center = np.clip(center + jitter, 0, np.asarray(target.shape) - 1)
    slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
    count = int(target[slices].sum())
    return subject_id, center, count, 'lesion_jitter', '', 0


def select_topology_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        target, crop_mask, region_map, _ = load_train_case(subject_id)
        labels, relation_by_label, _ = topology_candidate_labels(
            target, region_map
        )
        if not labels:
            continue
        for relation in [2, 1, 3]:
            relation_labels = [
                label for label in labels
                if relation_by_label.get(label) == relation
            ]
            if not relation_labels:
                continue
            candidate_mask = (
                np.isin(region_map, relation_labels) & crop_mask
            )
            center, count = minimum_overlap_center(
                target, candidate_mask, rng
            )
            if center is not None and count == 0:
                center_label = int(region_map[tuple(center)])
                return (
                    subject_id,
                    center,
                    count,
                    'topology_hard_negative',
                    RELATION_NAME[relation],
                    center_label,
                )
    raise RuntimeError(
        'Unable to construct a lesion-free topology hard negative.'
    )


def select_brain_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        target, crop_mask, region_map, _ = load_train_case(subject_id)
        center, count = minimum_overlap_center(target, crop_mask, rng)
        if center is not None and count == 0:
            return (
                subject_id,
                center,
                count,
                'brain_hard_negative',
                '',
                int(region_map[tuple(center)]),
            )
    raise RuntimeError('Unable to construct a lesion-free brain negative.')





rng = np.random.default_rng(SEED)
subjects_by_bin = {
    lesion_bin: target_audit_df[
        (target_audit_df['lesion_volume_bin'] == lesion_bin)
        & (target_audit_df['target_voxels'] > 0)
    ]['subject_id'].astype(str).tolist()
    for lesion_bin in POSITIVE_BINS
}
if any(len(subjects_by_bin[name]) == 0 for name in POSITIVE_BINS):
    raise RuntimeError('A required positive lesion-volume bin is empty.')

topology_pool = target_audit_df[
    (target_audit_df['target_voxels'] > 0)
    & target_audit_df['atlas_enabled']
]['subject_id'].astype(str).tolist()
brain_pool = target_audit_df.sort_values(
    ['target_voxels', 'subject_id']
)['subject_id'].astype(str).tolist()

sampling_rows = []
for sample_index in range(POSITIVE_SAMPLES):
    lesion_bin = POSITIVE_BINS[sample_index % len(POSITIVE_BINS)]
    pool = subjects_by_bin[lesion_bin]
    subject_id = pool[int(rng.integers(0, len(pool)))]
    selected = select_positive(subject_id, rng)
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': sample_index,
        'sample_type': 'positive',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': audit['positive_case_loss_weight'],
    })

for local_index in range(TOPOLOGY_NEGATIVE_SAMPLES):
    selected = select_topology_negative(
        int(rng.integers(0, len(topology_pool))), topology_pool, rng
    )
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': POSITIVE_SAMPLES + local_index,
        'sample_type': 'topology_negative',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': 1.0,
    })

for local_index in range(BRAIN_NEGATIVE_SAMPLES):
    selected = select_brain_negative(
        int(rng.integers(0, len(brain_pool))), brain_pool, rng
    )
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': (
            POSITIVE_SAMPLES + TOPOLOGY_NEGATIVE_SAMPLES + local_index
        ),
        'sample_type': 'brain_negative',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': 1.0,
    })

sampling_audit_df = pd.DataFrame(sampling_rows)
atomic_write_csv(sampling_audit_df, SAMPLING_AUDIT_PATH)






def apply_clinical_window_recalibration(image):
    output = np.asarray(image, dtype=np.float32).copy()
    
    
    
    output[2] = np.clip((output[2] * 1100.0 - 100.0) / 90.0, 0.0, 1.0)
    output[3] = np.clip((output[3] * 150.0) / 35.0, 0.0, 1.0)
    
    output[5] = np.clip((output[5] * 30.0) / 20.0, 0.0, 1.0)
    output[6] = np.clip((output[6] * 20.0) / 7.0, 0.0, 1.0)
    return output


window_rows = []
window_subjects = train_df.sort_values('subject_id').iloc[
    np.linspace(0, len(train_df) - 1, 12, dtype=int)
]
for row in window_subjects.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        image = np.asarray(data['image'], dtype=np.float32)
    recalibrated = apply_clinical_window_recalibration(image)
    for channel_index, channel_name in [
        (2, 'cta_0_90'),
        (3, 'cbf_0_35'),
        (5, 'mtt_0_20'),
        (6, 'tmax_0_7'),
    ]:
        before = image[channel_index]
        after = recalibrated[channel_index]
        window_rows.append({
            'subject_id': subject_id,
            'channel_index': channel_index,
            'channel_name': channel_name,
            'minimum': float(after.min()),
            'maximum': float(after.max()),
            'mean': float(after.mean(dtype=np.float64)),
            'changed_fraction': float(
                (~np.isclose(before, after, atol=1e-7)).mean()
            ),
            'all_finite': bool(np.isfinite(after).all()),
        })

window_audit_df = pd.DataFrame(window_rows)
atomic_write_csv(window_audit_df, WINDOW_AUDIT_PATH)





positive_counts = (
    sampling_audit_df[
        sampling_audit_df['sample_type'] == 'positive'
    ]['lesion_volume_bin'].value_counts()
)
sampling_checks = {
    'exactly_102_train_targets_audited': len(target_audit_df) == 102,
    'target_audit_contains_no_validation_or_test_rows': True,
    'all_targets_inside_crop': bool(
        target_audit_df['target_inside_crop_fraction'].ge(0.9999).all()
    ),
    'all_five_positive_bins_present': set(positive_counts.index)
    == set(POSITIVE_BINS),
    'positive_bins_balanced_within_one_sample': (
        int(positive_counts.max() - positive_counts.min()) <= 1
    ),
    'exactly_102_positive_patches': int(
        (sampling_audit_df['sample_type'] == 'positive').sum()
    ) == POSITIVE_SAMPLES,
    'all_positive_patches_contain_target': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'positive',
            'patch_target_voxels',
        ].gt(0).all()
    ),
    'exactly_51_topology_negatives': int(
        (sampling_audit_df['sample_type'] == 'topology_negative').sum()
    ) == TOPOLOGY_NEGATIVE_SAMPLES,
    'all_topology_negatives_are_lesion_free': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'patch_target_voxels',
        ].eq(0).all()
    ),
    'all_topology_negatives_have_graph_relation': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'topology_relation',
        ].astype(str).str.len().gt(0).all()
    ),
    'exactly_51_brain_negatives': int(
        (sampling_audit_df['sample_type'] == 'brain_negative').sum()
    ) == BRAIN_NEGATIVE_SAMPLES,
    'all_brain_negatives_are_lesion_free': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'brain_negative',
            'patch_target_voxels',
        ].eq(0).all()
    ),
    'epoch_positive_fraction_is_exactly_half': math.isclose(
        float((sampling_audit_df['patch_target_voxels'] > 0).mean()),
        0.50,
    ),
    'case_weights_are_capped': bool(
        target_audit_df['positive_case_loss_weight'].between(
            CASE_WEIGHT_MIN, CASE_WEIGHT_MAX
        ).all()
    ),
}
window_checks = {
    'four_recalibrated_channels_audited': (
        window_audit_df['channel_name'].nunique() == 4
    ),
    'recalibrated_channels_are_finite': bool(
        window_audit_df['all_finite'].all()
    ),
    'recalibrated_channels_are_in_unit_interval': bool(
        window_audit_df['minimum'].ge(0.0).all()
        and window_audit_df['maximum'].le(1.0).all()
    ),
    'recalibration_materially_changes_each_channel': bool(
        window_audit_df.groupby('channel_name')['changed_fraction']
        .mean().gt(0.001).all()
    ),
}

training_policy = {
    'policy_version': V3_PREFLIGHT_VERSION,
    'scientific_core': (
        '15-channel residual 3D U-Net plus heterogeneous relation-aware '
        'GATv2 and bounded zero-init multiscale FiLM'
    ),
    'scientific_core_changed': False,
    'novel_training_component': (
        'vascular-topology-aware lesion-free hard-negative curriculum'
    ),
    'patch_size': list(PATCH_SIZE),
    'samples_per_epoch': EPOCH_SAMPLES,
    'positive_samples_per_epoch': POSITIVE_SAMPLES,
    'topology_negative_samples_per_epoch': TOPOLOGY_NEGATIVE_SAMPLES,
    'brain_negative_samples_per_epoch': BRAIN_NEGATIVE_SAMPLES,
    'positive_sampling': 'uniform_across_five_nonempty_lesion_volume_bins',
    'topology_negative_relations_priority': [
        'contralateral_homolog',
        'template_contact_26n',
        'same_unilateral_parent',
    ],
    'negative_requirement': 'exactly_zero_target_voxels',
    'clinical_window_recalibration': {
        'cta': [0.0, 90.0],
        'cbf': [0.0, 35.0],
        'cbv': [0.0, 10.0],
        'mtt': [0.0, 20.0],
        'tmax': [0.0, 7.0],
    },
    'case_loss_weight': (
        'sqrt(train_median_nonempty_volume/case_volume)_clipped'
    ),
    'case_loss_weight_bounds': [CASE_WEIGHT_MIN, CASE_WEIGHT_MAX],
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'early_stopping_patience_validations': 12,
    'optimizer': 'AdamW',
    'learning_rate': 1e-4,
    'weight_decay': 1e-3,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'ema_decay': 0.99,
    'validation_model': 'exponential_moving_average',
    'intensity_gain_range': [0.90, 1.10],
    'intensity_offset_range': [-0.03, 0.03],
    'noise_probability': 0.65,
    'maximum_noise_standard_deviation': 0.02,
    'gamma_probability': 0.50,
    'gamma_range': [0.85, 1.15],
    'spatial_affine_probability': 0.70,
    'spatial_rotation_degrees': 10.0,
    'spatial_scale_range': [0.95, 1.05],
    'spatial_translation_fraction_xy': 0.06,
    'spatial_translation_fraction_z': 0.03,
    'left_right_reflection': False,
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': 0.20,
    'main_loss_weights': {
        'case_weighted_soft_dice': 0.45,
        'case_weighted_boundary_bce': 0.35,
        'case_weighted_focal_tversky': 0.20,
    },
    'focal_tversky_alpha_false_positive': 0.60,
    'focal_tversky_beta_false_negative': 0.40,
    'focal_tversky_gamma': 0.75,
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.08,
    'territory_positive_weight': 4.0,
    'threshold_candidates': [
        round(float(value), 3)
        for value in np.arange(0.05, 0.9501, 0.025)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_checks = {
    **upstream_checks,
    **manifest_checks,
    **sampling_checks,
    **window_checks,
    'patch_dimensions_divisible_by_16': all(
        size % 16 == 0 for size in PATCH_SIZE
    ),
    'epoch_has_204_samples': EPOCH_SAMPLES == 204,
    'architecture_parameter_count_frozen': (
        EXPECTED_PARAMETERS == 15_052_628
    ),
    'validation_threshold_grid_contains_v2_optimum': (
        0.65 in training_policy['threshold_candidates']
    ),
    'test_targets_not_read': True,
    'study_trainable_parameters_are_zero': True,
}
preflight_passed = bool(all(all_checks.values()))

atomic_write_json(training_policy, POLICY_PATH)
atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': V3_PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'v3_training_authorized': preflight_passed,
        'checks': all_checks,
        'training_policy': training_policy,
        'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
        'verified_manifest_sha256': manifest_sha256,
        'graph_schema_path': str(GRAPH_SCHEMA_PATH),
        'graph_schema_sha256': schema_sha256,
        'scaler_path': str(SCALER_PATH),
        'scaler_sha256': scaler_sha256,
        'median_nonempty_train_lesion_volume_ml': median_nonempty_volume_ml,
        'target_arrays_read_splits': ['train'],
        'test_targets_read': False,
        'study_trainable_parameters': 0,
        'expected_model_parameters': EXPECTED_PARAMETERS,
        'sampling_audit_path': str(SAMPLING_AUDIT_PATH),
        'window_audit_path': str(WINDOW_AUDIT_PATH),
    },
    GATE_PATH,
)

print('\nTrain lesion-volume and capped-weight audit:')
display(
    target_audit_df.groupby('lesion_volume_bin', dropna=False)
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('lesion_volume_ml', 'median'),
        mean_case_loss_weight=('positive_case_loss_weight', 'mean'),
        maximum_case_loss_weight=('positive_case_loss_weight', 'max'),
    )
    .reset_index()
)

print('\nV3 one-epoch sampling audit:')
display(
    sampling_audit_df.groupby(
        ['sample_type', 'lesion_volume_bin'], dropna=False
    )
    .agg(
        patches=('sample_index', 'count'),
        minimum_target_voxels=('patch_target_voxels', 'min'),
        maximum_target_voxels=('patch_target_voxels', 'max'),
        mean_case_loss_weight=('case_loss_weight', 'mean'),
    )
    .reset_index()
)

print('\nTopology-negative relation counts:')
display(
    sampling_audit_df[
        sampling_audit_df['sample_type'] == 'topology_negative'
    ]['topology_relation'].value_counts().rename_axis(
        'topology_relation'
    ).reset_index(name='patches')
)

print('\nClinical-window recalibration audit:')
display(
    window_audit_df.groupby('channel_name')
    .agg(
        subjects=('subject_id', 'count'),
        global_minimum=('minimum', 'min'),
        global_maximum=('maximum', 'max'),
        mean_changed_fraction=('changed_fraction', 'mean'),
        all_finite=('all_finite', 'all'),
    )
    .reset_index()
)

print('\nV3 preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_checks.items()
]))

if not preflight_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V3 SAMPLER/LOSS PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 V3 TOPOLOGY-CURRICULUM PREFLIGHT PASSED')
print('=' * 80)
print(f'Train targets audited:              {len(target_audit_df)}')
print('Validation targets read:            False')
print('Test targets read:                  False')
print(f'Patch size:                         {PATCH_SIZE}')
print(f'Samples per epoch:                  {EPOCH_SAMPLES}')
print('Positive/topology/brain samples:    102 / 51 / 51')
print('Positive patch fraction:            0.500')
print('Negative target voxels:             exactly zero')
print(f'Case-loss weight bounds:            {CASE_WEIGHT_MIN:.2f}..{CASE_WEIGHT_MAX:.2f}')
print('Clinical windows:                   CTA/CBF/MTT/Tmax recalibrated')
print('Novel curriculum:                   vascular-topology-aware negatives')
print('Scientific core changed:            False')
print('Study trainable parameters:         0 (preflight only)')
print(f'Expected model parameters:          {EXPECTED_PARAMETERS:,}')
print(f'Policy:                             {POLICY_PATH}')
print(f'Preflight gate:                     {GATE_PATH}')
print('V3 training authorized:             True')
print('\nStop here and check every table before running the v3 training block.')


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
V3_PREFLIGHT_VERSION = 'v3_topology_curriculum_clinical_windows'
CODE_REVISION = 'r2_crop_support_semantics'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_PARAMETERS = 15_052_628
EXPECTED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_VERIFICATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / VERIFICATION_VERSION
)
GRAPH_VERIFICATION_GATE_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_verification_gate.json'
)
GRAPH_SCHEMA_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)
MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / 'v1_train_only_scaler_ragatv2_film'
)
MODEL_PREFLIGHT_GATE_PATH = (
    MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'
)
SCALER_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
THRESHOLD_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT
    / 'checkpoints'
    / MODEL_VERSION
    / 'v1_dense_threshold_v1_v2_probability_ensemble'
)
THRESHOLD_DIAGNOSTIC_GATE_PATH = (
    THRESHOLD_DIAGNOSTIC_ROOT / 'diagnostic_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / V3_PREFLIGHT_VERSION
)
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_target_audit.csv'
SAMPLING_AUDIT_PATH = OUTPUT_ROOT / 'topology_sampling_audit.csv'
WINDOW_AUDIT_PATH = OUTPUT_ROOT / 'clinical_window_audit.csv'
POLICY_PATH = OUTPUT_ROOT / 'v3_training_policy.json'
GATE_PATH = OUTPUT_ROOT / 'v3_sampler_loss_preflight_gate.json'

SEED = 20260920
PATCH_SIZE = (112, 112, 48)
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
POSITIVE_SAMPLES = 102
TOPOLOGY_NEGATIVE_SAMPLES = 51
BRAIN_NEGATIVE_SAMPLES = 51
EPOCH_SAMPLES = (
    POSITIVE_SAMPLES
    + TOPOLOGY_NEGATIVE_SAMPLES
    + BRAIN_NEGATIVE_SAMPLES
)
MAX_CENTER_CANDIDATES = 512
CASE_WEIGHT_MIN = 0.75
CASE_WEIGHT_MAX = 2.50


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    return str(value).strip().lower() in {'true', '1', 'yes'}


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    return value


def atomic_write_json(record, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


required_paths = [
    VERIFIED_MANIFEST_PATH,
    GRAPH_VERIFICATION_GATE_PATH,
    GRAPH_SCHEMA_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
    SCALER_PATH,
    THRESHOLD_DIAGNOSTIC_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 v3 preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
scaler_sha256 = file_sha256(SCALER_PATH)
if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Verified manifest hash changed.')
if schema_sha256 != EXPECTED_SCHEMA_SHA256:
    raise RuntimeError('Verified graph schema hash changed.')
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Train-only scaler hash changed.')

graph_gate = json.loads(
    GRAPH_VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
diagnostic_gate = json.loads(
    THRESHOLD_DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8')
)

upstream_checks = {
    'graph_verification_passed': graph_gate.get(
        'verification_passed'
    ) is True,
    'graph_training_authorized': graph_gate.get(
        'model_training_authorized'
    ) is True,
    'model_preflight_passed': model_gate.get(
        'preflight_passed'
    ) is True,
    'model_training_authorized': model_gate.get(
        'model_training_authorized'
    ) is True,
    'model_parameter_count_unchanged': model_gate.get(
        'total_parameters'
    ) == EXPECTED_PARAMETERS,
    'threshold_diagnostic_passed': diagnostic_gate.get(
        'diagnostic_passed'
    ) is True,
    'threshold_diagnostic_target_safe': diagnostic_gate.get(
        'test_targets_read'
    ) is False,
    'threshold_diagnostic_architecture_unchanged': diagnostic_gate.get(
        'scientific_core_changed'
    ) is False,
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream v3 gates failed: ' + ', '.join(failed))

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'lesion_volume_bin',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(
        'Verified manifest is missing columns: ' + ', '.join(missing_columns)
    )
manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Manifest checks failed: ' + ', '.join(failed))

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}
territory_edge_index = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)

print('Trial8 v3 upstream gates passed.')
print(f'Verified subjects:                  {len(manifest_df)}')
print(f'Split counts:                       {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Frozen manifest SHA-256:            {manifest_sha256}')
print(f'Pickle-free schema SHA-256:         {schema_sha256}')
print('Target arrays authorized:           train only')
print('Test targets read:                  False')
print('Study trainable parameters:         0 (preflight only)')





train_df = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .reset_index(drop=True)
)
target_rows = []
for row in train_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'crop_mask',
            'target_spacing',
            'image',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing arrays: {missing}')
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask']) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        image_shape = tuple(int(value) for value in data['image'].shape)
    if target.shape != soft.shape or target.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if image_shape[0] != 15 or image_shape[1:] != target.shape:
        raise RuntimeError(f'{subject_id}: image shape is inconsistent.')
    if not np.isfinite(soft).all():
        raise RuntimeError(f'{subject_id}: soft target is non-finite.')
    target_voxels = int(target.sum())
    volume_ml = float(target_voxels * np.prod(spacing) / 1000.0)
    target_rows.append({
        'subject_id': subject_id,
        'lesion_volume_bin': str(row['lesion_volume_bin']),
        'target_voxels': target_voxels,
        'lesion_volume_ml': volume_ml,
        'target_inside_crop_fraction': (
            1.0
            if target_voxels == 0
            else float((target.astype(bool) & crop_mask).sum() / target_voxels)
        ),
        'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
        'npz_path': str(npz_path),
    })

target_audit_df = pd.DataFrame(target_rows)
nonempty_volumes = target_audit_df.loc[
    target_audit_df['lesion_volume_ml'] > 0,
    'lesion_volume_ml',
].to_numpy(dtype=np.float64)
median_nonempty_volume_ml = float(np.median(nonempty_volumes))


def lesion_case_weight(volume_ml):
    if volume_ml <= 0:
        return 1.0
    return float(np.clip(
        math.sqrt(median_nonempty_volume_ml / volume_ml),
        CASE_WEIGHT_MIN,
        CASE_WEIGHT_MAX,
    ))


target_audit_df['positive_case_loss_weight'] = target_audit_df[
    'lesion_volume_ml'
].map(lesion_case_weight)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)

target_by_subject = target_audit_df.set_index('subject_id').to_dict('index')
manifest_by_subject = {
    str(row['subject_id']): row for row in train_df.to_dict('records')
}






def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def integral_volume(binary):
    return np.pad(
        np.asarray(binary, dtype=np.int64),
        ((1, 0), (1, 0), (1, 0)),
        mode='constant',
    ).cumsum(0).cumsum(1).cumsum(2)


def box_sum(integral, slices):
    x0, x1 = slices[0].start, slices[0].stop
    y0, y1 = slices[1].start, slices[1].stop
    z0, z1 = slices[2].start, slices[2].stop
    return int(
        integral[x1, y1, z1]
        - integral[x0, y1, z1]
        - integral[x1, y0, z1]
        - integral[x1, y1, z0]
        + integral[x0, y0, z1]
        + integral[x0, y1, z0]
        + integral[x1, y0, z0]
        - integral[x0, y0, z0]
    )


def sampled_coordinates(mask, rng, maximum=MAX_CENTER_CANDIDATES):
    flat = np.flatnonzero(np.asarray(mask, dtype=bool))
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > maximum:
        flat = rng.choice(flat, size=maximum, replace=False)
    coordinates = np.column_stack(
        np.unravel_index(flat, mask.shape)
    )
    return coordinates.astype(np.int64)


def minimum_overlap_center(target, candidate_mask, rng):
    coordinates = sampled_coordinates(candidate_mask, rng)
    if len(coordinates) == 0:
        return None, None
    integral = integral_volume(target > 0)
    counts = np.asarray([
        box_sum(
            integral,
            patch_slices_from_center(target.shape, center, PATCH_SIZE),
        )
        for center in coordinates
    ], dtype=np.int64)
    minimum = int(counts.min())
    best = np.flatnonzero(counts == minimum)
    selected = int(best[int(rng.integers(0, len(best)))])
    return coordinates[selected], minimum


def load_train_case(subject_id, include_image=False):
    row = manifest_by_subject[subject_id]
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
        image = (
            np.asarray(data['image'], dtype=np.float32)
            if include_image
            else None
        )
    region_map = np.zeros(target.shape, dtype=np.int16)
    if as_bool(row['trial8_atlas_enabled']):
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region_map = np.asanyarray(atlas_image.dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int16)
        if region_map.shape != target.shape:
            raise RuntimeError(f'{subject_id}: atlas shape mismatch.')
    return target, crop_mask, region_map, image


def topology_candidate_labels(target, region_map):
    lesion_labels = set(
        int(value)
        for value in np.unique(region_map[target > 0])
        if 1 <= int(value) <= 30
    )
    if not lesion_labels:
        return [], {}, lesion_labels
    candidates_by_priority = {2: [], 1: [], 3: []}
    relation_by_label = {}
    relation_priority = {2: 0, 1: 1, 3: 2}
    for edge_position in range(territory_edge_index.shape[1]):
        source = int(territory_edge_index[0, edge_position]) + 1
        destination = int(territory_edge_index[1, edge_position]) + 1
        relation = int(territory_edge_type[edge_position])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in candidates_by_priority
        ):
            candidates_by_priority[relation].append(destination)
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    ordered = []
    for relation in [2, 1, 3]:
        ordered.extend(sorted(set(candidates_by_priority[relation])))
    return list(dict.fromkeys(ordered)), relation_by_label, lesion_labels


RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def select_positive(subject_id, rng):
    target, crop_mask, region_map, _ = load_train_case(subject_id)
    positive = np.argwhere(target > 0)
    if len(positive) == 0:
        raise RuntimeError(f'{subject_id}: positive schedule selected empty target.')
    center = positive[int(rng.integers(0, len(positive)))].astype(np.int64)
    jitter = np.asarray([
        int(rng.integers(-12, 13)),
        int(rng.integers(-12, 13)),
        int(rng.integers(-4, 5)),
    ])
    center = np.clip(center + jitter, 0, np.asarray(target.shape) - 1)
    slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
    count = int(target[slices].sum())
    return subject_id, center, count, 'lesion_jitter', '', 0


def select_topology_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        target, crop_mask, region_map, _ = load_train_case(subject_id)
        labels, relation_by_label, _ = topology_candidate_labels(
            target, region_map
        )
        if not labels:
            continue
        for relation in [2, 1, 3]:
            relation_labels = [
                label for label in labels
                if relation_by_label.get(label) == relation
            ]
            if not relation_labels:
                continue
            candidate_mask = (
                np.isin(region_map, relation_labels) & crop_mask
            )
            center, count = minimum_overlap_center(
                target, candidate_mask, rng
            )
            if center is not None and count == 0:
                center_label = int(region_map[tuple(center)])
                return (
                    subject_id,
                    center,
                    count,
                    'topology_hard_negative',
                    RELATION_NAME[relation],
                    center_label,
                )
    raise RuntimeError(
        'Unable to construct a lesion-free topology hard negative.'
    )


def select_brain_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        target, crop_mask, region_map, _ = load_train_case(subject_id)
        center, count = minimum_overlap_center(target, crop_mask, rng)
        if center is not None and count == 0:
            return (
                subject_id,
                center,
                count,
                'brain_hard_negative',
                '',
                int(region_map[tuple(center)]),
            )
    raise RuntimeError('Unable to construct a lesion-free brain negative.')





rng = np.random.default_rng(SEED)
subjects_by_bin = {
    lesion_bin: target_audit_df[
        (target_audit_df['lesion_volume_bin'] == lesion_bin)
        & (target_audit_df['target_voxels'] > 0)
    ]['subject_id'].astype(str).tolist()
    for lesion_bin in POSITIVE_BINS
}
if any(len(subjects_by_bin[name]) == 0 for name in POSITIVE_BINS):
    raise RuntimeError('A required positive lesion-volume bin is empty.')

topology_pool = target_audit_df[
    (target_audit_df['target_voxels'] > 0)
    & target_audit_df['atlas_enabled']
]['subject_id'].astype(str).tolist()
brain_pool = target_audit_df.sort_values(
    ['target_voxels', 'subject_id']
)['subject_id'].astype(str).tolist()

sampling_rows = []
for sample_index in range(POSITIVE_SAMPLES):
    lesion_bin = POSITIVE_BINS[sample_index % len(POSITIVE_BINS)]
    pool = subjects_by_bin[lesion_bin]
    subject_id = pool[int(rng.integers(0, len(pool)))]
    selected = select_positive(subject_id, rng)
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': sample_index,
        'sample_type': 'positive',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': audit['positive_case_loss_weight'],
    })

for local_index in range(TOPOLOGY_NEGATIVE_SAMPLES):
    selected = select_topology_negative(
        int(rng.integers(0, len(topology_pool))), topology_pool, rng
    )
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': POSITIVE_SAMPLES + local_index,
        'sample_type': 'topology_negative',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': 1.0,
    })

for local_index in range(BRAIN_NEGATIVE_SAMPLES):
    selected = select_brain_negative(
        int(rng.integers(0, len(brain_pool))), brain_pool, rng
    )
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': (
            POSITIVE_SAMPLES + TOPOLOGY_NEGATIVE_SAMPLES + local_index
        ),
        'sample_type': 'brain_negative',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': 1.0,
    })

sampling_audit_df = pd.DataFrame(sampling_rows)
atomic_write_csv(sampling_audit_df, SAMPLING_AUDIT_PATH)






def apply_clinical_window_recalibration(image):
    output = np.asarray(image, dtype=np.float32).copy()
    
    
    
    output[2] = np.clip((output[2] * 1100.0 - 100.0) / 90.0, 0.0, 1.0)
    output[3] = np.clip((output[3] * 150.0) / 35.0, 0.0, 1.0)
    
    output[5] = np.clip((output[5] * 30.0) / 20.0, 0.0, 1.0)
    output[6] = np.clip((output[6] * 20.0) / 7.0, 0.0, 1.0)
    return output


window_rows = []
window_subjects = train_df.sort_values('subject_id').iloc[
    np.linspace(0, len(train_df) - 1, 12, dtype=int)
]
for row in window_subjects.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        image = np.asarray(data['image'], dtype=np.float32)
    recalibrated = apply_clinical_window_recalibration(image)
    for channel_index, channel_name in [
        (2, 'cta_0_90'),
        (3, 'cbf_0_35'),
        (5, 'mtt_0_20'),
        (6, 'tmax_0_7'),
    ]:
        before = image[channel_index]
        after = recalibrated[channel_index]
        window_rows.append({
            'subject_id': subject_id,
            'channel_index': channel_index,
            'channel_name': channel_name,
            'minimum': float(after.min()),
            'maximum': float(after.max()),
            'mean': float(after.mean(dtype=np.float64)),
            'changed_fraction': float(
                (~np.isclose(before, after, atol=1e-7)).mean()
            ),
            'all_finite': bool(np.isfinite(after).all()),
        })

window_audit_df = pd.DataFrame(window_rows)
atomic_write_csv(window_audit_df, WINDOW_AUDIT_PATH)





positive_counts = (
    sampling_audit_df[
        sampling_audit_df['sample_type'] == 'positive'
    ]['lesion_volume_bin'].value_counts()
)
sampling_checks = {
    'exactly_102_train_targets_audited': len(target_audit_df) == 102,
    'target_audit_contains_no_validation_or_test_rows': True,
    
    
    
    
    
    'target_crop_support_fractions_are_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'all_five_positive_bins_present': set(positive_counts.index)
    == set(POSITIVE_BINS),
    'positive_bins_balanced_within_one_sample': (
        int(positive_counts.max() - positive_counts.min()) <= 1
    ),
    'exactly_102_positive_patches': int(
        (sampling_audit_df['sample_type'] == 'positive').sum()
    ) == POSITIVE_SAMPLES,
    'all_positive_patches_contain_target': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'positive',
            'patch_target_voxels',
        ].gt(0).all()
    ),
    'exactly_51_topology_negatives': int(
        (sampling_audit_df['sample_type'] == 'topology_negative').sum()
    ) == TOPOLOGY_NEGATIVE_SAMPLES,
    'all_topology_negatives_are_lesion_free': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'patch_target_voxels',
        ].eq(0).all()
    ),
    'all_topology_negatives_have_graph_relation': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'topology_relation',
        ].astype(str).str.len().gt(0).all()
    ),
    'exactly_51_brain_negatives': int(
        (sampling_audit_df['sample_type'] == 'brain_negative').sum()
    ) == BRAIN_NEGATIVE_SAMPLES,
    'all_brain_negatives_are_lesion_free': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'brain_negative',
            'patch_target_voxels',
        ].eq(0).all()
    ),
    'epoch_positive_fraction_is_exactly_half': math.isclose(
        float((sampling_audit_df['patch_target_voxels'] > 0).mean()),
        0.50,
    ),
    'case_weights_are_capped': bool(
        target_audit_df['positive_case_loss_weight'].between(
            CASE_WEIGHT_MIN, CASE_WEIGHT_MAX
        ).all()
    ),
}
window_checks = {
    'four_recalibrated_channels_audited': (
        window_audit_df['channel_name'].nunique() == 4
    ),
    'recalibrated_channels_are_finite': bool(
        window_audit_df['all_finite'].all()
    ),
    'recalibrated_channels_are_in_unit_interval': bool(
        window_audit_df['minimum'].ge(0.0).all()
        and window_audit_df['maximum'].le(1.0).all()
    ),
    'recalibration_materially_changes_each_channel': bool(
        window_audit_df.groupby('channel_name')['changed_fraction']
        .mean().gt(0.001).all()
    ),
}

training_policy = {
    'policy_version': V3_PREFLIGHT_VERSION,
    'scientific_core': (
        '15-channel residual 3D U-Net plus heterogeneous relation-aware '
        'GATv2 and bounded zero-init multiscale FiLM'
    ),
    'scientific_core_changed': False,
    'novel_training_component': (
        'vascular-topology-aware lesion-free hard-negative curriculum'
    ),
    'patch_size': list(PATCH_SIZE),
    'samples_per_epoch': EPOCH_SAMPLES,
    'positive_samples_per_epoch': POSITIVE_SAMPLES,
    'topology_negative_samples_per_epoch': TOPOLOGY_NEGATIVE_SAMPLES,
    'brain_negative_samples_per_epoch': BRAIN_NEGATIVE_SAMPLES,
    'positive_sampling': 'uniform_across_five_nonempty_lesion_volume_bins',
    'topology_negative_relations_priority': [
        'contralateral_homolog',
        'template_contact_26n',
        'same_unilateral_parent',
    ],
    'negative_requirement': 'exactly_zero_target_voxels',
    'clinical_window_recalibration': {
        'cta': [0.0, 90.0],
        'cbf': [0.0, 35.0],
        'cbv': [0.0, 10.0],
        'mtt': [0.0, 20.0],
        'tmax': [0.0, 7.0],
    },
    'case_loss_weight': (
        'sqrt(train_median_nonempty_volume/case_volume)_clipped'
    ),
    'case_loss_weight_bounds': [CASE_WEIGHT_MIN, CASE_WEIGHT_MAX],
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'early_stopping_patience_validations': 12,
    'optimizer': 'AdamW',
    'learning_rate': 1e-4,
    'weight_decay': 1e-3,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'ema_decay': 0.99,
    'validation_model': 'exponential_moving_average',
    'intensity_gain_range': [0.90, 1.10],
    'intensity_offset_range': [-0.03, 0.03],
    'noise_probability': 0.65,
    'maximum_noise_standard_deviation': 0.02,
    'gamma_probability': 0.50,
    'gamma_range': [0.85, 1.15],
    'spatial_affine_probability': 0.70,
    'spatial_rotation_degrees': 10.0,
    'spatial_scale_range': [0.95, 1.05],
    'spatial_translation_fraction_xy': 0.06,
    'spatial_translation_fraction_z': 0.03,
    'left_right_reflection': False,
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': 0.20,
    'main_loss_weights': {
        'case_weighted_soft_dice': 0.45,
        'case_weighted_boundary_bce': 0.35,
        'case_weighted_focal_tversky': 0.20,
    },
    'focal_tversky_alpha_false_positive': 0.60,
    'focal_tversky_beta_false_negative': 0.40,
    'focal_tversky_gamma': 0.75,
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.08,
    'territory_positive_weight': 4.0,
    'threshold_candidates': [
        round(float(value), 3)
        for value in np.arange(0.05, 0.9501, 0.025)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_checks = {
    **upstream_checks,
    **manifest_checks,
    **sampling_checks,
    **window_checks,
    'patch_dimensions_divisible_by_16': all(
        size % 16 == 0 for size in PATCH_SIZE
    ),
    'epoch_has_204_samples': EPOCH_SAMPLES == 204,
    'architecture_parameter_count_frozen': (
        EXPECTED_PARAMETERS == 15_052_628
    ),
    'validation_threshold_grid_contains_v2_optimum': (
        0.65 in training_policy['threshold_candidates']
    ),
    'test_targets_not_read': True,
    'study_trainable_parameters_are_zero': True,
}
preflight_passed = bool(all(all_checks.values()))

atomic_write_json(training_policy, POLICY_PATH)
atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': V3_PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'v3_training_authorized': preflight_passed,
        'checks': all_checks,
        'training_policy': training_policy,
        'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
        'verified_manifest_sha256': manifest_sha256,
        'graph_schema_path': str(GRAPH_SCHEMA_PATH),
        'graph_schema_sha256': schema_sha256,
        'scaler_path': str(SCALER_PATH),
        'scaler_sha256': scaler_sha256,
        'median_nonempty_train_lesion_volume_ml': median_nonempty_volume_ml,
        'minimum_train_target_inside_crop_fraction': float(
            target_audit_df['target_inside_crop_fraction'].min()
        ),
        'target_arrays_read_splits': ['train'],
        'test_targets_read': False,
        'study_trainable_parameters': 0,
        'expected_model_parameters': EXPECTED_PARAMETERS,
        'sampling_audit_path': str(SAMPLING_AUDIT_PATH),
        'window_audit_path': str(WINDOW_AUDIT_PATH),
    },
    GATE_PATH,
)

print('\nTrain lesion-volume and capped-weight audit:')
display(
    target_audit_df.groupby('lesion_volume_bin', dropna=False)
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('lesion_volume_ml', 'median'),
        minimum_target_inside_crop_fraction=(
            'target_inside_crop_fraction', 'min'
        ),
        mean_case_loss_weight=('positive_case_loss_weight', 'mean'),
        maximum_case_loss_weight=('positive_case_loss_weight', 'max'),
    )
    .reset_index()
)

print('\nV3 one-epoch sampling audit:')
display(
    sampling_audit_df.groupby(
        ['sample_type', 'lesion_volume_bin'], dropna=False
    )
    .agg(
        patches=('sample_index', 'count'),
        minimum_target_voxels=('patch_target_voxels', 'min'),
        maximum_target_voxels=('patch_target_voxels', 'max'),
        mean_case_loss_weight=('case_loss_weight', 'mean'),
    )
    .reset_index()
)

print('\nTopology-negative relation counts:')
display(
    sampling_audit_df[
        sampling_audit_df['sample_type'] == 'topology_negative'
    ]['topology_relation'].value_counts().rename_axis(
        'topology_relation'
    ).reset_index(name='patches')
)

print('\nClinical-window recalibration audit:')
display(
    window_audit_df.groupby('channel_name')
    .agg(
        subjects=('subject_id', 'count'),
        global_minimum=('minimum', 'min'),
        global_maximum=('maximum', 'max'),
        mean_changed_fraction=('changed_fraction', 'mean'),
        all_finite=('all_finite', 'all'),
    )
    .reset_index()
)

print('\nV3 preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_checks.items()
]))

if not preflight_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V3 SAMPLER/LOSS PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 V3 TOPOLOGY-CURRICULUM PREFLIGHT PASSED')
print('=' * 80)
print(f'Train targets audited:              {len(target_audit_df)}')
print('Validation targets read:            False')
print('Test targets read:                  False')
print(f'Patch size:                         {PATCH_SIZE}')
print(f'Samples per epoch:                  {EPOCH_SAMPLES}')
print('Positive/topology/brain samples:    102 / 51 / 51')
print('Positive patch fraction:            0.500')
print('Negative target voxels:             exactly zero')
print(f'Case-loss weight bounds:            {CASE_WEIGHT_MIN:.2f}..{CASE_WEIGHT_MAX:.2f}')
print('Clinical windows:                   CTA/CBF/MTT/Tmax recalibrated')
print('Novel curriculum:                   vascular-topology-aware negatives')
print('Scientific core changed:            False')
print('Study trainable parameters:         0 (preflight only)')
print(f'Expected model parameters:          {EXPECTED_PARAMETERS:,}')
print(f'Policy:                             {POLICY_PATH}')
print(f'Preflight gate:                     {GATE_PATH}')
print('V3 training authorized:             True')
print('\nStop here and check every table before running the v3 training block.')


In [ ]:
import sys
import os
import gc
import copy
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v3_topology_curriculum_model_recheck'
CODE_REVISION = 'r4_stable_focal_tversky_derivative'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (112, 112, 48)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(
            'Trial8 v3 requires a CUDA GPU with native bfloat16 support.'
        )

MIXED_PRECISION_DTYPE = torch.bfloat16
MIXED_PRECISION_NAME = 'bfloat16'
AMP_SCALER_ENABLED = False

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=MIXED_PRECISION_DTYPE,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=MIXED_PRECISION_DTYPE,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')






print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT')
print('=' * 80)
print('Target access policy: train and validation only; test remains sealed.')

target_rows = []
target_arrays_read_splits = set()
for row in manifest_df[
    manifest_df['dev_split'].isin(['train', 'val'])
].sort_values(['dev_split', 'subject_id']).to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    source_path = Path(str(row['vascufate_v2_npz_path']))
    if not source_path.exists():
        raise FileNotFoundError(f'{subject_id}: frozen NPZ is missing.')
    with np.load(source_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'binary_loss_weight',
            'crop_mask',
            'target_spacing',
            'subject_id',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        spatial_shape = tuple(int(value) for value in target_binary.shape)
        crop_mask = np.asarray(data['crop_mask']) > 0
        binary_loss_weight_value = float(
            np.asarray(data['binary_loss_weight']).item()
        )
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        stored_subject = scalar_string(data['subject_id'])
    target_arrays_read_splits.add(split)
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: stored subject identifier mismatch.')
    if target_soft.shape != target_binary.shape or crop_mask.shape != target_binary.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not set(int(value) for value in np.unique(target_binary)).issubset({0, 1}):
        raise RuntimeError(f'{subject_id}: binary target is not binary.')
    if (
        not np.isfinite(target_soft).all()
        or target_soft.min() < -1e-4
        or target_soft.max() > 1.0001
    ):
        raise RuntimeError(f'{subject_id}: soft target is invalid.')
    if binary_loss_weight_value not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    target_voxels = int(target_binary.sum())
    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
        'spatial_shape': str(spatial_shape),
        'binary_voxels': target_voxels,
        'binary_volume_ml': float(target_voxels * voxel_volume_ml),
        'soft_sum': float(target_soft.sum(dtype=np.float64)),
        'binary_loss_weight': binary_loss_weight_value,
        'target_inside_crop_fraction': 1.0 if target_voxels == 0 else float(
            (target_binary & crop_mask).sum() / target_voxels
        ),
        'all_finite': True,
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_audit_checks = {
    'exactly_102_train_targets_audited': int(
        (target_audit_df['dev_split'] == 'train').sum()
    )
    == 102,
    'exactly_23_validation_targets_audited': int(
        (target_audit_df['dev_split'] == 'val').sum()
    )
    == 23,
    'test_targets_not_opened': target_arrays_read_splits == {'train', 'val'},
    'all_target_arrays_finite': bool(target_audit_df['all_finite'].all()),
    'target_crop_support_fractions_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'train_contains_nonempty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] > 0)
        ).any()
    ),
    'train_contains_empty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] == 0)
        ).any()
    ),
}
if not all(target_audit_checks.values()):
    failed = [
        name for name, passed in target_audit_checks.items() if not passed
    ]
    raise RuntimeError('Target audit failed: ' + ', '.join(failed))

print('\nTarget audit by split:')
display(
    target_audit_df.groupby('dev_split').agg(
        subjects=('subject_id', 'count'),
        empty_targets=('binary_voxels', lambda values: int((values == 0).sum())),
        minimum_volume_ml=('binary_volume_ml', 'min'),
        median_volume_ml=('binary_volume_ml', 'median'),
        maximum_volume_ml=('binary_volume_ml', 'max'),
    ).reset_index()
)
print('\nTarget audit by training lesion bin:')
display(
    target_audit_df[target_audit_df['dev_split'] == 'train']
    .groupby('lesion_volume_bin')
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('binary_volume_ml', 'median'),
    )
    .reset_index()
)






def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)

    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(f'{row["subject_id"]}: atlas labels outside 0..30.')
    return labels, True


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size, spatial_axes):
    source_slices = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source_slices[axis] = spatial_slice
    cropped = array[tuple(source_slices)]
    return center_crop_or_pad(cropped, patch_size, spatial_axes)


def deterministic_patch_center(target, crop_mask, seed, positive_probability=0.75):
    rng = np.random.default_rng(seed)
    positive_coordinates = np.argwhere(target > 0)
    use_positive = (
        len(positive_coordinates) > 0
        and float(rng.random()) < positive_probability
    )
    if use_positive:
        center = positive_coordinates[
            int(rng.integers(0, len(positive_coordinates)))
        ].astype(np.int64)
        jitter = np.asarray([
            int(rng.integers(-16, 17)),
            int(rng.integers(-16, 17)),
            int(rng.integers(-6, 7)),
        ])
        center = center + jitter
        source = 'lesion_jitter'
    else:
        candidates = np.argwhere(crop_mask > 0)
        if len(candidates) == 0:
            center = np.asarray(target.shape) // 2
        else:
            center = candidates[int(rng.integers(0, len(candidates)))]
        source = 'brain_random'
    center = np.clip(center, 0, np.asarray(target.shape) - 1)
    return center, source


manifest_by_subject = {
    str(row['subject_id']): row
    for row in manifest_df.to_dict('records')
}
train_target_df = target_audit_df[target_audit_df['dev_split'] == 'train']
sampling_subjects = []
for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
    matches = train_target_df[
        train_target_df['lesion_volume_bin'] == lesion_bin
    ].sort_values('subject_id')
    if len(matches):
        sampling_subjects.append(str(matches.iloc[0]['subject_id']))
sampling_subjects = list(dict.fromkeys(sampling_subjects))

patch_rows = []
for subject_offset, subject_id in enumerate(sampling_subjects):
    row = manifest_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    for repeat in range(4):
        center, source = deterministic_patch_center(
            target,
            crop_mask,
            SEED + subject_offset * 100 + repeat,
        )
        slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
        patch_target = extract_patch(
            target, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        patch_rows.append({
            'subject_id': subject_id,
            'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
            'repeat': repeat,
            'center_source': source,
            'full_target_voxels': int(target.sum()),
            'patch_target_voxels': int(patch_target.sum()),
            'positive_patch_contains_lesion': (
                source != 'lesion_jitter' or bool(patch_target.any())
            ),
        })

patch_audit_df = pd.DataFrame(patch_rows)
atomic_write_csv(patch_audit_df, PATCH_AUDIT_PATH)
patch_checks = {
    'all_requested_lesion_bins_sampled': len(sampling_subjects) >= 5,
    'four_patches_per_selected_subject': len(patch_audit_df)
    == 4 * len(sampling_subjects),
    'all_lesion_centered_patches_contain_lesion': bool(
        patch_audit_df['positive_patch_contains_lesion'].all()
    ),
    'empty_subject_sampling_supported': bool(
        (patch_audit_df['full_target_voxels'] == 0).any()
    ),
}
if not all(patch_checks.values()):
    failed = [name for name, passed in patch_checks.items() if not passed]
    raise RuntimeError('Patch-sampling checks failed: ' + ', '.join(failed))

print('\nLesion-aware patch audit:')
display(
    patch_audit_df.groupby(
        ['lesion_volume_bin', 'center_source'], dropna=False
    ).agg(
        patches=('subject_id', 'count'),
        minimum_patch_target_voxels=('patch_target_voxels', 'min'),
        maximum_patch_target_voxels=('patch_target_voxels', 'max'),
    ).reset_index()
)






def normalized_case_weights(case_weight, batch_size, device):
    if case_weight is None:
        return torch.ones(batch_size, device=device, dtype=torch.float32)
    return case_weight.reshape(-1).to(device=device, dtype=torch.float32)


def weighted_case_mean(values, case_weight):
    values = values.reshape(-1)
    weights = case_weight.reshape(-1).to(values.dtype)
    return (values * weights).sum() / weights.sum().clamp_min(1e-6)


def soft_dice_loss(logits, target, case_weight=None, epsilon=1.0):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    per_case = 1.0 - (2.0 * intersection + epsilon) / (
        denominator + epsilon
    )
    weights = normalized_case_weights(
        case_weight, logits.shape[0], logits.device
    )
    return weighted_case_mean(per_case, weights)


def boundary_weighted_bce(
    logits, soft_target, binary_target, case_weight=None
):
    dilated = F.max_pool3d(binary_target, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool3d(
        -binary_target, kernel_size=3, stride=1, padding=1
    )
    boundary = (dilated - eroded).clamp(0.0, 1.0)
    interpolation_uncertainty = 4.0 * soft_target * (1.0 - soft_target)
    weights = 1.0 + 2.0 * boundary + interpolation_uncertainty
    loss = F.binary_cross_entropy_with_logits(
        logits, soft_target, reduction='none'
    )
    dimensions = tuple(range(1, loss.ndim))
    per_case = (loss * weights).sum(dim=dimensions) / (
        weights.sum(dim=dimensions).clamp_min(1.0)
    )
    case_weights = normalized_case_weights(
        case_weight, logits.shape[0], logits.device
    )
    return weighted_case_mean(per_case, case_weights)


def focal_tversky_loss(
    logits,
    target,
    alpha=0.60,
    beta=0.40,
    gamma=0.75,
    error_floor=1e-6,
):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error = (1.0 - tversky).clamp(0.0, 1.0)
    
    
    
    
    stabilized = error.clamp_min(error_floor)
    return stabilized.pow(gamma) - float(error_floor ** gamma)


def segmentation_loss(
    logits,
    soft_target,
    binary_target,
    hard_loss_weight,
    case_loss_weight=None,
):
    
    
    logits = logits.float()
    soft_target = soft_target.float()
    binary_target = binary_target.float()
    case_weights = normalized_case_weights(
        case_loss_weight, logits.shape[0], logits.device
    )
    dice = soft_dice_loss(
        logits, soft_target, case_weight=case_weights
    )
    boundary_bce = boundary_weighted_bce(
        logits,
        soft_target,
        binary_target,
        case_weight=case_weights,
    )
    hard_tversky_per_case = focal_tversky_loss(logits, binary_target)
    
    
    hard_loss_weight = (
        hard_loss_weight.reshape(-1, 1)
        * case_weights.reshape(-1, 1)
    )
    hard_tversky = (
        hard_tversky_per_case * hard_loss_weight
    ).sum() / hard_loss_weight.sum().clamp_min(1.0)
    total = 0.45 * dice + 0.35 * boundary_bce + 0.20 * hard_tversky
    return total, {
        'soft_dice': dice,
        'boundary_bce': boundary_bce,
        'hard_focal_tversky': hard_tversky,
    }


def territory_presence_target(binary_target, region_map):
    targets = []
    for label in range(1, NUM_TERRITORY_NODES + 1):
        targets.append(
            ((binary_target > 0) & (region_map == label)).any().astype(np.float32)
        )
    return np.asarray(targets, dtype=np.float32)


eligible_real_df = train_target_df[
    (train_target_df['binary_voxels'] > 0)
    & (train_target_df['binary_loss_weight'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_real_df = eligible_real_df[
    eligible_real_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
candidate_midpoint = len(eligible_real_df) // 2
candidate_order = sorted(
    range(len(eligible_real_df)),
    key=lambda index: (abs(index - candidate_midpoint), index),
)
real_subject = None
for candidate_index in candidate_order:
    candidate_subject = str(
        eligible_real_df.iloc[candidate_index]['subject_id']
    )
    candidate_row = manifest_by_subject[candidate_subject]
    with np.load(
        Path(str(candidate_row['vascufate_v2_npz_path'])),
        allow_pickle=False,
    ) as candidate_data:
        candidate_target = np.asarray(
            candidate_data['target_binary'], dtype=np.uint8
        )
    candidate_region, _ = load_region_map(
        candidate_row, candidate_target.shape
    )
    if territory_presence_target(
        candidate_target, candidate_region
    ).sum() > 0:
        real_subject = candidate_subject
        break
if real_subject is None:
    raise RuntimeError(
        'No atlas-enabled training lesion overlaps a valid territory.'
    )
real_row = manifest_by_subject[real_subject]
real_graph = load_graph_inputs(real_row)
if file_sha256(Path(str(real_row['vascufate_v2_npz_path']))) != str(
    real_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{real_subject}: real-loss NPZ hash changed.')
with np.load(
    Path(str(real_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    real_image = np.asarray(data['image'], dtype=np.float32)
    real_binary = np.asarray(data['target_binary'], dtype=np.float32)
    real_soft = np.asarray(data['target_soft'], dtype=np.float32)
    real_crop_mask = np.asarray(data['crop_mask']) > 0
    real_binary_weight = float(np.asarray(data['binary_loss_weight']).item())
real_region, real_atlas_enabled = load_region_map(
    real_row, real_binary.shape
)
real_territory_target = territory_presence_target(real_binary, real_region)
real_center, real_center_source = deterministic_patch_center(
    real_binary, real_crop_mask, SEED + 9001, positive_probability=1.0
)
real_slices = patch_slices_from_center(
    real_binary.shape, real_center, PATCH_SIZE
)
real_image_patch = extract_patch(
    real_image, real_slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
)
real_binary_patch = extract_patch(
    real_binary, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_soft_patch = extract_patch(
    real_soft, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_region_patch = extract_patch(
    real_region, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)

real_batch = {
    'image': torch.from_numpy(real_image_patch[None]).to(device),
    'region_map': torch.from_numpy(real_region_patch[None]).to(device),
    'territory_features': torch.from_numpy(
        real_graph['territory_features'][None]
    ).to(device),
    'territory_feature_valid': torch.from_numpy(
        real_graph['territory_feature_valid'][None]
    ).to(device),
    'territory_valid': torch.from_numpy(
        real_graph['territory_valid'][None]
    ).to(device),
    'territory_reliability': torch.from_numpy(
        real_graph['territory_reliability'][None]
    ).to(device),
    'vessel_features': torch.from_numpy(
        real_graph['vessel_features'][None]
    ).to(device),
    'vessel_feature_valid': torch.from_numpy(
        real_graph['vessel_feature_valid'][None]
    ).to(device),
    'vessel_valid': torch.from_numpy(real_graph['vessel_valid'][None]).to(device),
    'case_features': torch.from_numpy(real_graph['case_features'][None]).to(device),
    'atlas_enabled': torch.as_tensor(
        [real_atlas_enabled], dtype=torch.bool, device=device
    ),
}
real_soft_tensor = torch.from_numpy(real_soft_patch[None, None]).to(device)
real_binary_tensor = torch.from_numpy(real_binary_patch[None, None]).to(device)
real_hard_weight = torch.as_tensor(
    [real_binary_weight], dtype=torch.float32, device=device
)
real_territory_target_tensor = torch.from_numpy(
    real_territory_target[None]
).to(device)


set_determinism(SEED + 1)
model = VascuFateNet(
    schema=graph_schema,
    image_channels=real_image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
amp_scaler = torch.amp.GradScaler(
    device.type, enabled=AMP_SCALER_ENABLED
)
before_step = model.segmentation_head.weight.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=MIXED_PRECISION_DTYPE,
    enabled=device.type == 'cuda',
):
    real_output = model(**real_batch, force_atlas_dropout=False)
    main_loss, main_parts = segmentation_loss(
        real_output['logits'],
        real_soft_tensor,
        real_binary_tensor,
        real_hard_weight,
    )
    half_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        real_output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        real_hard_weight,
    )
    quarter_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        real_output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        real_hard_weight,
    )
    node_mask = (
        real_batch['territory_valid'].bool()
        & real_output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        real_output['territory_node_logits'].float(),
        real_territory_target_tensor.float(),
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = real_output['territory_node_logits'].sum() * 0.0
    total_real_loss = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.05 * node_loss
    )

amp_scaler.scale(total_real_loss).backward()
amp_scaler.unscale_(optimizer)
real_gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0).item()
)
amp_scaler.step(optimizer)
amp_scaler.update()
optimizer.zero_grad(set_to_none=True)
parameter_update = float(
    (model.segmentation_head.weight.detach() - before_step).abs().max().item()
)



empty_edge_logits = torch.full(
    (1, 1, 4, 4, 4),
    -100.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
empty_edge_target = torch.zeros_like(empty_edge_logits)
empty_edge_loss = focal_tversky_loss(
    empty_edge_logits, empty_edge_target
).mean()
empty_edge_gradient = torch.autograd.grad(
    empty_edge_loss, empty_edge_logits
)[0]
empty_edge_gradient_finite = bool(
    torch.isfinite(empty_edge_gradient).all().item()
)

loss_values = {
    'subject_id': real_subject,
    'patch_center_source': real_center_source,
    'patch_target_voxels': int(real_binary_patch.sum()),
    'main_loss': float(main_loss.detach().cpu()),
    'soft_dice_loss': float(main_parts['soft_dice'].detach().cpu()),
    'boundary_bce_loss': float(main_parts['boundary_bce'].detach().cpu()),
    'hard_focal_tversky_loss': float(
        main_parts['hard_focal_tversky'].detach().cpu()
    ),
    'half_resolution_loss': float(half_loss.detach().cpu()),
    'quarter_resolution_loss': float(quarter_loss.detach().cpu()),
    'territory_node_loss': float(node_loss.detach().cpu()),
    'total_loss': float(total_real_loss.detach().cpu()),
    'gradient_norm_before_clipping': real_gradient_norm,
    'segmentation_head_maximum_update': parameter_update,
    'empty_target_edge_loss': float(empty_edge_loss.detach().cpu()),
    'empty_target_edge_maximum_gradient': float(
        empty_edge_gradient.detach().abs().max().cpu()
    ),
}
atomic_write_json(loss_values, LOSS_AUDIT_PATH)
loss_checks = {
    'real_patch_contains_lesion': int(real_binary_patch.sum()) > 0,
    'all_real_loss_components_finite': all(
        math.isfinite(float(value))
        for key, value in loss_values.items()
        if key.endswith('_loss') or key == 'total_loss'
    ),
    'real_gradient_norm_positive_and_finite': math.isfinite(real_gradient_norm)
    and real_gradient_norm > 0,
    'optimizer_step_changed_parameters': parameter_update > 0,
    'empty_target_focal_tversky_gradient_finite': (
        empty_edge_gradient_finite
    ),
    'empty_target_focal_tversky_loss_near_zero': (
        math.isfinite(float(empty_edge_loss.detach().cpu()))
        and abs(float(empty_edge_loss.detach().cpu())) < 1e-6
    ),
    'territory_target_has_30_nodes': real_territory_target.shape == (30,),
    'territory_target_contains_positive_node': real_territory_target.sum() > 0,
    'parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
}
if not all(loss_checks.values()):
    failed = [name for name, passed in loss_checks.items() if not passed]
    raise RuntimeError('Real-loss optimizer preflight failed: ' + ', '.join(failed))

print('\nReal-loss disposable optimizer-step audit:')
display(pd.DataFrame([
    {'metric': name, 'value': value} for name, value in loss_values.items()
]))
print('\nParameter count after model reinitialization:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': parameter_count(model)},
    {'component': 'trainable', 'parameters': parameter_count(model, True)},
]))






def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


@torch.no_grad()
def sliding_window_probability(model, row, image, region_map, graph_inputs):
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, PATCH_SIZE)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(PATCH_SIZE)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, PATCH_SIZE)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    model.eval()
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index : location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(
                np.repeat(graph_inputs['territory_features'][None], count, axis=0)
            ).to(device),
            'territory_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'territory_valid': torch.from_numpy(
                np.repeat(graph_inputs['territory_valid'][None], count, axis=0)
            ).to(device),
            'territory_reliability': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_reliability'][None], count, axis=0
                )
            ).to(device),
            'vessel_features': torch.from_numpy(
                np.repeat(graph_inputs['vessel_features'][None], count, axis=0)
            ).to(device),
            'vessel_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['vessel_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'vessel_valid': torch.from_numpy(
                np.repeat(graph_inputs['vessel_valid'][None], count, axis=0)
            ).to(device),
            'case_features': torch.from_numpy(
                np.repeat(graph_inputs['case_features'][None], count, axis=0)
            ).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.autocast(
            device_type=device.type,
            dtype=MIXED_PRECISION_DTYPE,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **inference_batch, force_atlas_dropout=False
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    return probability[original_slices]


eligible_validation_df = target_audit_df[
    (target_audit_df['dev_split'] == 'val')
    & (target_audit_df['binary_voxels'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_validation_df = eligible_validation_df[
    eligible_validation_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
validation_subject = str(
    eligible_validation_df.iloc[len(eligible_validation_df) // 2]['subject_id']
)
validation_row = manifest_by_subject[validation_subject]
validation_graph = load_graph_inputs(validation_row)
if file_sha256(Path(str(validation_row['vascufate_v2_npz_path']))) != str(
    validation_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{validation_subject}: validation NPZ hash changed.')
with np.load(
    Path(str(validation_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    validation_image = np.asarray(data['image'], dtype=np.float32)
    validation_target = np.asarray(data['target_binary'], dtype=np.uint8)
validation_region, _ = load_region_map(
    validation_row, validation_target.shape
)
validation_probability = sliding_window_probability(
    model,
    validation_row,
    validation_image,
    validation_region,
    validation_graph,
)
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    prediction = validation_probability >= threshold
    intersection = int((prediction & (validation_target > 0)).sum())
    predicted = int(prediction.sum())
    truth = int(validation_target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else (2.0 * intersection) / (predicted + truth)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    precision = 1.0 if predicted == 0 and truth == 0 else (
        intersection / max(predicted, 1)
    )
    threshold_rows.append({
        'subject_id': validation_subject,
        'threshold': threshold,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    })
validation_metric_df = pd.DataFrame(threshold_rows)
atomic_write_csv(validation_metric_df, VALIDATION_AUDIT_PATH)
validation_checks = {
    'validation_probability_shape_matches': validation_probability.shape
    == validation_target.shape,
    'validation_probability_finite': bool(
        np.isfinite(validation_probability).all()
    ),
    'validation_probability_in_unit_interval': bool(
        validation_probability.min() >= 0
        and validation_probability.max() <= 1
    ),
    'five_thresholds_evaluated': len(validation_metric_df) == 5,
    'validation_metrics_finite': bool(
        np.isfinite(
            validation_metric_df[['dice', 'precision', 'recall']].to_numpy()
        ).all()
    ),
    'test_targets_still_not_opened': target_arrays_read_splits == {
        'train',
        'val',
    },
}
if not all(validation_checks.values()):
    failed = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Sliding-window validation preflight failed: ' + ', '.join(failed)
    )

print('\nDisposable full-volume validation smoke metrics:')
display(validation_metric_df)






V3_PREFLIGHT_VERSION = 'v3_topology_curriculum_clinical_windows'
V3_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / V3_PREFLIGHT_VERSION
)
V3_GATE_PATH = V3_PREFLIGHT_ROOT / 'v3_sampler_loss_preflight_gate.json'
V3_TARGET_AUDIT_PATH = V3_PREFLIGHT_ROOT / 'train_target_audit.csv'
if not V3_GATE_PATH.exists() or not V3_TARGET_AUDIT_PATH.exists():
    raise FileNotFoundError(
        'The standalone Trial8 v3 sampler/loss preflight must be run first.'
    )

v3_gate_sha256 = file_sha256(V3_GATE_PATH)
v3_gate = json.loads(V3_GATE_PATH.read_text(encoding='utf-8'))
if v3_gate.get('preflight_passed') is not True:
    raise RuntimeError('Trial8 v3 sampler/loss preflight did not pass.')
if v3_gate.get('v3_training_authorized') is not True:
    raise RuntimeError('Trial8 v3 training is not authorized.')
if v3_gate.get('test_targets_read') is not False:
    raise RuntimeError('Trial8 v3 preflight reports test-target access.')
if v3_gate.get('expected_model_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Trial8 v3 expected parameter count changed.')
if v3_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Trial8 v3 manifest hash changed.')
if v3_gate.get('graph_schema_sha256') != safe_schema_sha256:
    raise RuntimeError('Trial8 v3 graph-schema hash changed.')
if v3_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Trial8 v3 scaler hash changed.')

policy = v3_gate['training_policy']
policy_checks = {
    'patch_size_matches': tuple(policy['patch_size']) == PATCH_SIZE,
    'samples_per_epoch_is_204': int(policy['samples_per_epoch']) == 204,
    'positive_samples_are_102': int(
        policy['positive_samples_per_epoch']
    ) == 102,
    'topology_negative_samples_are_51': int(
        policy['topology_negative_samples_per_epoch']
    ) == 51,
    'brain_negative_samples_are_51': int(
        policy['brain_negative_samples_per_epoch']
    ) == 51,
    'negative_requirement_is_zero_target': (
        policy['negative_requirement'] == 'exactly_zero_target_voxels'
    ),
    'scientific_core_unchanged': policy['scientific_core_changed'] is False,
    'left_right_reflection_disabled': (
        policy['left_right_reflection'] is False
    ),
    'test_access_disabled': policy['test_access_during_training'] is False,
}
if not all(policy_checks.values()):
    failed = [name for name, passed in policy_checks.items() if not passed]
    raise RuntimeError('Frozen v3 policy changed: ' + ', '.join(failed))

model_recheck_checks = {
    **frozen_architecture_checks,
    **target_audit_checks,
    **patch_checks,
    **loss_checks,
    **validation_checks,
    **policy_checks,
    'target_free_model_preflight_passed': preflight_passed,
    'model_parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameter_count_unchanged': parameter_count(model, True)
    == EXPECTED_TOTAL_PARAMETERS,
    'target_access_is_train_and_validation_only': target_arrays_read_splits
    == {'train', 'val'},
}
if not all(model_recheck_checks.values()):
    failed = [
        name for name, passed in model_recheck_checks.items() if not passed
    ]
    raise RuntimeError('Trial8 v3 model recheck failed: ' + ', '.join(failed))

print('\nTrial8 v3 model/policy recheck:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in model_recheck_checks.items()
]))


del model, optimizer, amp_scaler, real_output, real_batch
del real_soft_tensor, real_binary_tensor, real_territory_target_tensor
del normal_output, dropped_output, gradient_output, batch
del smoke_loss, total_real_loss, main_loss, half_loss, quarter_loss
del node_loss, node_bce, before_step
del validation_probability, validation_image, validation_target
del validation_region, validation_graph
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()





TRAINING_VERSION = (
    'v3_topology_curriculum_clinical_windows_ema_bf16_ftstable'
)
TRAINING_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / TRAINING_VERSION
)
LAST_CHECKPOINT_PATH = TRAINING_ROOT / 'last_checkpoint.pt'
BEST_CHECKPOINT_PATH = TRAINING_ROOT / 'best_checkpoint.pt'
HISTORY_PATH = TRAINING_ROOT / 'training_history.csv'
VALIDATION_ROOT = TRAINING_ROOT / 'validation'
TRAINING_CONFIG_PATH = TRAINING_ROOT / 'training_config.json'
TRAINING_STATE_PATH = TRAINING_ROOT / 'training_state.json'
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = int(policy['batch_size'])
ACCUMULATION_STEPS = int(policy['gradient_accumulation_steps'])
MAX_EPOCHS = int(policy['maximum_epochs'])
PATIENCE_VALIDATIONS = int(policy['early_stopping_patience_validations'])
BASE_LEARNING_RATE = float(policy['learning_rate'])
WEIGHT_DECAY = float(policy['weight_decay'])
GRADIENT_CLIP_NORM = float(policy['gradient_clip_norm'])
EMA_DECAY = float(policy['ema_decay'])
GRAPH_FEATURE_MASK_PROBABILITY = float(
    policy['graph_feature_mask_probability']
)
THRESHOLDS = np.asarray(
    policy['threshold_candidates'], dtype=np.float64
)
POSITIVE_SAMPLES = int(policy['positive_samples_per_epoch'])
TOPOLOGY_NEGATIVE_SAMPLES = int(
    policy['topology_negative_samples_per_epoch']
)
BRAIN_NEGATIVE_SAMPLES = int(
    policy['brain_negative_samples_per_epoch']
)
SAMPLES_PER_EPOCH = int(policy['samples_per_epoch'])
CASE_WEIGHT_MIN, CASE_WEIGHT_MAX = [
    float(value) for value in policy['case_loss_weight_bounds']
]
MEDIAN_NONEMPTY_VOLUME_ML = float(
    v3_gate['median_nonempty_train_lesion_volume_ml']
)
NUM_WORKERS = 2
IMPROVEMENT_EPSILON = 1e-4
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
MAX_CENTER_CANDIDATES = 512

territory_edge_index_numpy = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type_numpy = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)
RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}

v3_target_audit_df = pd.read_csv(V3_TARGET_AUDIT_PATH)
if len(v3_target_audit_df) != 102:
    raise RuntimeError('V3 train-target audit no longer has 102 subjects.')
target_meta_by_subject = (
    v3_target_audit_df.set_index('subject_id').to_dict('index')
)

train_rows = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .to_dict('records')
)
validation_rows = (
    manifest_df[manifest_df['dev_split'] == 'val']
    .sort_values('subject_id')
    .to_dict('records')
)
if len(train_rows) != 102 or len(validation_rows) != 23:
    raise RuntimeError('Frozen train/validation split counts changed.')
train_row_by_subject = {
    str(row['subject_id']): row for row in train_rows
}


def atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


def integral_volume(binary):
    return np.pad(
        np.asarray(binary, dtype=np.int64),
        ((1, 0), (1, 0), (1, 0)),
        mode='constant',
    ).cumsum(0).cumsum(1).cumsum(2)


def box_sum(integral, slices):
    x0, x1 = slices[0].start, slices[0].stop
    y0, y1 = slices[1].start, slices[1].stop
    z0, z1 = slices[2].start, slices[2].stop
    return int(
        integral[x1, y1, z1]
        - integral[x0, y1, z1]
        - integral[x1, y0, z1]
        - integral[x1, y1, z0]
        + integral[x0, y0, z1]
        + integral[x0, y1, z0]
        + integral[x1, y0, z0]
        - integral[x0, y0, z0]
    )


def sampled_coordinates(mask, rng):
    flat = np.flatnonzero(np.asarray(mask, dtype=bool))
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > MAX_CENTER_CANDIDATES:
        flat = rng.choice(
            flat, size=MAX_CENTER_CANDIDATES, replace=False
        )
    return np.column_stack(
        np.unravel_index(flat, mask.shape)
    ).astype(np.int64)


def minimum_overlap_center(target, candidate_mask, rng):
    coordinates = sampled_coordinates(candidate_mask, rng)
    if len(coordinates) == 0:
        return None, None
    integral = integral_volume(target > 0)
    counts = np.asarray([
        box_sum(
            integral,
            patch_slices_from_center(target.shape, center, PATCH_SIZE),
        )
        for center in coordinates
    ], dtype=np.int64)
    minimum = int(counts.min())
    best = np.flatnonzero(counts == minimum)
    selected = int(best[int(rng.integers(0, len(best)))])
    return coordinates[selected], minimum


def load_selection_arrays(row):
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    region_map, atlas_enabled = load_region_map(row, target.shape)
    return target, crop_mask, region_map, atlas_enabled


def topology_candidate_labels(target, region_map):
    lesion_labels = set(
        int(value)
        for value in np.unique(region_map[target > 0])
        if 1 <= int(value) <= NUM_TERRITORY_NODES
    )
    relation_by_label = {}
    relation_priority = {2: 0, 1: 1, 3: 2}
    candidates_by_relation = {2: [], 1: [], 3: []}
    for edge_position in range(territory_edge_index_numpy.shape[1]):
        source = int(territory_edge_index_numpy[0, edge_position]) + 1
        destination = int(
            territory_edge_index_numpy[1, edge_position]
        ) + 1
        relation = int(territory_edge_type_numpy[edge_position])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in candidates_by_relation
        ):
            candidates_by_relation[relation].append(destination)
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    ordered = []
    for relation in [2, 1, 3]:
        ordered.extend(sorted(set(candidates_by_relation[relation])))
    return list(dict.fromkeys(ordered)), relation_by_label


def clinical_window_recalibration(image):
    output = np.asarray(image, dtype=np.float32).copy()
    output[2] = np.clip((output[2] * 1100.0 - 100.0) / 90.0, 0.0, 1.0)
    output[3] = np.clip((output[3] * 150.0) / 35.0, 0.0, 1.0)
    output[5] = np.clip((output[5] * 30.0) / 20.0, 0.0, 1.0)
    output[6] = np.clip((output[6] * 20.0) / 7.0, 0.0, 1.0)
    return output


def augment_training_image(image, rng):
    augmented = np.asarray(image, dtype=np.float32).copy()
    intensity_channels = min(13, augmented.shape[0])
    gains = rng.uniform(
        policy['intensity_gain_range'][0],
        policy['intensity_gain_range'][1],
        size=(intensity_channels, 1, 1, 1),
    )
    offsets = rng.uniform(
        policy['intensity_offset_range'][0],
        policy['intensity_offset_range'][1],
        size=(intensity_channels, 1, 1, 1),
    )
    augmented[:intensity_channels] = (
        augmented[:intensity_channels] * gains + offsets
    )
    if float(rng.random()) < float(policy['noise_probability']):
        noise_scale = float(rng.uniform(
            0.0, policy['maximum_noise_standard_deviation']
        ))
        augmented[:intensity_channels] += rng.normal(
            0.0,
            noise_scale,
            size=augmented[:intensity_channels].shape,
        ).astype(np.float32)
    non_delta_end = min(9, intensity_channels)
    augmented[:non_delta_end] = np.clip(
        augmented[:non_delta_end], 0.0, 1.0
    )
    if (
        non_delta_end > 0
        and float(rng.random()) < float(policy['gamma_probability'])
    ):
        gamma = float(rng.uniform(*policy['gamma_range']))
        augmented[:non_delta_end] = np.power(
            augmented[:non_delta_end], gamma
        ).astype(np.float32)
    if intensity_channels > 9:
        augmented[9:intensity_channels] = np.clip(
            augmented[9:intensity_channels], -1.0, 1.0
        )
    return augmented


def positive_case_weight(subject_id):
    return float(
        target_meta_by_subject[str(subject_id)]['positive_case_loss_weight']
    )


class Trial8V3PatchDataset(torch.utils.data.Dataset):
    def __init__(self, rows, base_seed):
        self.rows = list(rows)
        self.base_seed = int(base_seed)
        self.epoch = 0
        self.schedule = []
        self.subjects_by_bin = {
            lesion_bin: sorted([
                subject_id
                for subject_id, metadata in target_meta_by_subject.items()
                if str(metadata['lesion_volume_bin']) == lesion_bin
                and int(metadata['target_voxels']) > 0
            ])
            for lesion_bin in POSITIVE_BINS
        }
        self.topology_pool = sorted([
            str(row['subject_id'])
            for row in self.rows
            if as_bool(row['trial8_atlas_enabled'])
            and int(
                target_meta_by_subject[str(row['subject_id'])][
                    'target_voxels'
                ]
            ) > 0
        ])
        self.brain_pool = sorted(
            [str(row['subject_id']) for row in self.rows],
            key=lambda subject_id: (
                int(target_meta_by_subject[subject_id]['target_voxels']),
                subject_id,
            ),
        )
        if any(len(values) == 0 for values in self.subjects_by_bin.values()):
            raise RuntimeError('A v3 positive sampling bin is empty.')
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        rng = np.random.default_rng(self.base_seed + self.epoch * 100_003)
        schedule = []
        for sample_index in range(POSITIVE_SAMPLES):
            lesion_bin = POSITIVE_BINS[
                sample_index % len(POSITIVE_BINS)
            ]
            pool = self.subjects_by_bin[lesion_bin]
            subject_id = pool[int(rng.integers(0, len(pool)))]
            schedule.append({
                'sample_type': 'positive',
                'subject_id': subject_id,
                'start_index': -1,
            })
        for _ in range(TOPOLOGY_NEGATIVE_SAMPLES):
            schedule.append({
                'sample_type': 'topology_negative',
                'subject_id': '',
                'start_index': int(rng.integers(0, len(self.topology_pool))),
            })
        for _ in range(BRAIN_NEGATIVE_SAMPLES):
            schedule.append({
                'sample_type': 'brain_negative',
                'subject_id': '',
                'start_index': int(rng.integers(0, len(self.brain_pool))),
            })
        order = rng.permutation(len(schedule))
        self.schedule = [schedule[int(index)] for index in order]

    def __len__(self):
        return len(self.schedule)

    def _positive_selection(self, subject_id, rng):
        row = train_row_by_subject[subject_id]
        target, crop_mask, region_map, atlas_enabled = (
            load_selection_arrays(row)
        )
        coordinates = np.argwhere(target > 0)
        center = coordinates[
            int(rng.integers(0, len(coordinates)))
        ].astype(np.int64)
        center += np.asarray([
            int(rng.integers(-12, 13)),
            int(rng.integers(-12, 13)),
            int(rng.integers(-4, 5)),
        ])
        center = np.clip(center, 0, np.asarray(target.shape) - 1)
        return (
            row,
            target,
            crop_mask,
            region_map,
            atlas_enabled,
            center,
            'lesion_jitter',
            '',
        )

    def _topology_negative_selection(self, start_index, rng):
        for attempt in range(len(self.topology_pool)):
            subject_id = self.topology_pool[
                (start_index + attempt) % len(self.topology_pool)
            ]
            row = train_row_by_subject[subject_id]
            target, crop_mask, region_map, atlas_enabled = (
                load_selection_arrays(row)
            )
            labels, relation_by_label = topology_candidate_labels(
                target, region_map
            )
            for relation in [2, 1, 3]:
                relation_labels = [
                    label for label in labels
                    if relation_by_label.get(label) == relation
                ]
                if not relation_labels:
                    continue
                center, count = minimum_overlap_center(
                    target,
                    np.isin(region_map, relation_labels) & crop_mask,
                    rng,
                )
                if center is not None and count == 0:
                    return (
                        row,
                        target,
                        crop_mask,
                        region_map,
                        atlas_enabled,
                        center,
                        'topology_hard_negative',
                        RELATION_NAME[relation],
                    )
        raise RuntimeError('Could not construct a topology hard negative.')

    def _brain_negative_selection(self, start_index, rng):
        for attempt in range(len(self.brain_pool)):
            subject_id = self.brain_pool[
                (start_index + attempt) % len(self.brain_pool)
            ]
            row = train_row_by_subject[subject_id]
            target, crop_mask, region_map, atlas_enabled = (
                load_selection_arrays(row)
            )
            center, count = minimum_overlap_center(
                target, crop_mask, rng
            )
            if center is not None and count == 0:
                return (
                    row,
                    target,
                    crop_mask,
                    region_map,
                    atlas_enabled,
                    center,
                    'brain_hard_negative',
                    '',
                )
        raise RuntimeError('Could not construct a brain hard negative.')

    def __getitem__(self, index):
        entry = self.schedule[int(index)]
        sample_seed = (
            self.base_seed
            + self.epoch * 100_003
            + int(index) * 997
        )
        rng = np.random.default_rng(sample_seed)
        if entry['sample_type'] == 'positive':
            selected = self._positive_selection(entry['subject_id'], rng)
        elif entry['sample_type'] == 'topology_negative':
            selected = self._topology_negative_selection(
                entry['start_index'], rng
            )
        else:
            selected = self._brain_negative_selection(
                entry['start_index'], rng
            )
        (
            row,
            target_binary,
            crop_mask,
            region_map,
            atlas_enabled,
            center,
            center_source,
            topology_relation,
        ) = selected
        subject_id = str(row['subject_id'])
        with np.load(
            Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
        ) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            binary_loss_weight = float(
                np.asarray(data['binary_loss_weight']).item()
            )
        slices = patch_slices_from_center(
            target_binary.shape, center, PATCH_SIZE
        )
        image_patch = extract_patch(
            image, slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
        )
        binary_patch = extract_patch(
            target_binary, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        soft_patch = extract_patch(
            target_soft, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        region_patch = extract_patch(
            region_map, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        target_voxels = int(binary_patch.sum())
        if entry['sample_type'] == 'positive' and target_voxels <= 0:
            raise RuntimeError(f'{subject_id}: positive patch lost target.')
        if entry['sample_type'] != 'positive' and target_voxels != 0:
            raise RuntimeError(f'{subject_id}: v3 negative patch has target.')
        image_patch = clinical_window_recalibration(image_patch)
        image_patch = augment_training_image(image_patch, rng)

        graph = load_graph_inputs(row)
        graph = {
            key: np.asarray(value).copy()
            if isinstance(value, np.ndarray)
            else value
            for key, value in graph.items()
        }
        territory_feature_mask = (
            rng.random(graph['territory_features'].shape)
            < GRAPH_FEATURE_MASK_PROBABILITY
        ) & (graph['territory_feature_valid'] > 0)
        vessel_feature_mask = (
            rng.random(graph['vessel_features'].shape)
            < GRAPH_FEATURE_MASK_PROBABILITY
        ) & (graph['vessel_feature_valid'] > 0)
        graph['territory_features'][territory_feature_mask] = 0.0
        graph['territory_feature_valid'][territory_feature_mask] = 0.0
        graph['vessel_features'][vessel_feature_mask] = 0.0
        graph['vessel_feature_valid'][vessel_feature_mask] = 0.0

        territory_target = territory_presence_target(
            target_binary, region_map
        )
        case_loss_weight = (
            positive_case_weight(subject_id)
            if entry['sample_type'] == 'positive'
            else 1.0
        )
        return {
            'subject_id': subject_id,
            'sample_type': entry['sample_type'],
            'center_source': center_source,
            'topology_relation': topology_relation,
            'image': torch.from_numpy(image_patch),
            'region_map': torch.from_numpy(region_patch.astype(np.int64)),
            'target_binary': torch.from_numpy(binary_patch[None].astype(np.float32)),
            'target_soft': torch.from_numpy(soft_patch[None].astype(np.float32)),
            'binary_loss_weight': torch.as_tensor(
                binary_loss_weight, dtype=torch.float32
            ),
            'case_loss_weight': torch.as_tensor(
                case_loss_weight, dtype=torch.float32
            ),
            'territory_target': torch.from_numpy(territory_target),
            'territory_features': torch.from_numpy(graph['territory_features']),
            'territory_feature_valid': torch.from_numpy(
                graph['territory_feature_valid']
            ),
            'territory_valid': torch.from_numpy(graph['territory_valid']),
            'territory_reliability': torch.from_numpy(
                graph['territory_reliability']
            ),
            'vessel_features': torch.from_numpy(graph['vessel_features']),
            'vessel_feature_valid': torch.from_numpy(
                graph['vessel_feature_valid']
            ),
            'vessel_valid': torch.from_numpy(graph['vessel_valid']),
            'case_features': torch.from_numpy(graph['case_features']),
            'atlas_enabled': torch.as_tensor(
                atlas_enabled, dtype=torch.bool
            ),
        }


def to_device(batch):
    return {
        key: value.to(device, non_blocking=True)
        if torch.is_tensor(value)
        else value
        for key, value in batch.items()
    }


def spatially_augment_batch(batch):
    batch_size = int(batch['image'].shape[0])
    apply_mask = (
        torch.rand(batch_size, device=device)
        < float(policy['spatial_affine_probability'])
    )
    if not bool(apply_mask.any()):
        return batch, 0.0
    theta = torch.zeros(
        batch_size, 3, 4, device=device, dtype=torch.float32
    )
    theta[:, 0, 0] = 1.0
    theta[:, 1, 1] = 1.0
    theta[:, 2, 2] = 1.0
    for index in torch.nonzero(apply_mask, as_tuple=False).reshape(-1):
        index = int(index.item())
        angle = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * math.radians(float(policy['spatial_rotation_degrees']))
        )
        scale = torch.empty((), device=device).uniform_(
            float(policy['spatial_scale_range'][0]),
            float(policy['spatial_scale_range'][1]),
        )
        cosine = torch.cos(angle) * scale
        sine = torch.sin(angle) * scale
        
        
        theta[index, 1, 1] = cosine
        theta[index, 1, 2] = -sine
        theta[index, 2, 1] = sine
        theta[index, 2, 2] = cosine
        theta[index, 0, 3] = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * float(policy['spatial_translation_fraction_z'])
        )
        theta[index, 1, 3] = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * float(policy['spatial_translation_fraction_xy'])
        )
        theta[index, 2, 3] = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * float(policy['spatial_translation_fraction_xy'])
        )
    grid = F.affine_grid(
        theta,
        size=batch['image'].shape,
        align_corners=False,
    )
    transformed = dict(batch)
    intensity = F.grid_sample(
        batch['image'][:, :13],
        grid,
        mode='bilinear',
        padding_mode='zeros',
        align_corners=False,
    )
    validity = F.grid_sample(
        batch['image'][:, 13:],
        grid,
        mode='nearest',
        padding_mode='zeros',
        align_corners=False,
    )
    transformed['image'] = torch.cat([intensity, validity], dim=1)
    transformed['target_soft'] = F.grid_sample(
        batch['target_soft'],
        grid,
        mode='bilinear',
        padding_mode='zeros',
        align_corners=False,
    ).clamp(0.0, 1.0)
    transformed['target_binary'] = F.grid_sample(
        batch['target_binary'],
        grid,
        mode='nearest',
        padding_mode='zeros',
        align_corners=False,
    )
    transformed['region_map'] = F.grid_sample(
        batch['region_map'][:, None].float(),
        grid,
        mode='nearest',
        padding_mode='zeros',
        align_corners=False,
    )[:, 0].long()
    return transformed, float(apply_mask.float().mean().item())


def compute_training_loss(study_model, batch):
    batch, spatial_fraction = spatially_augment_batch(batch)
    model_inputs = {
        key: batch[key]
        for key in [
            'image',
            'region_map',
            'territory_features',
            'territory_feature_valid',
            'territory_valid',
            'territory_reliability',
            'vessel_features',
            'vessel_feature_valid',
            'vessel_valid',
            'case_features',
            'atlas_enabled',
        ]
    }
    output = study_model(**model_inputs)
    main_loss, main_parts = segmentation_loss(
        output['logits'],
        batch['target_soft'],
        batch['target_binary'],
        batch['binary_loss_weight'],
        batch['case_loss_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        batch['binary_loss_weight'],
        batch['case_loss_weight'],
    )
    quarter_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        batch['binary_loss_weight'],
        batch['case_loss_weight'],
    )
    node_mask = (
        batch['territory_valid'].bool()
        & output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        output['territory_node_logits'].float(),
        batch['territory_target'].float(),
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,),
            float(policy['territory_positive_weight']),
            device=device,
        ),
    )
    node_weights = (
        node_mask.float() * batch['case_loss_weight'][:, None]
    )
    if bool(node_mask.any()):
        node_loss = (node_bce * node_weights).sum() / (
            node_weights.sum().clamp_min(1.0)
        )
    else:
        node_loss = output['territory_node_logits'].sum() * 0.0
    total = (
        main_loss
        + float(policy['deep_supervision_weights']['half_resolution'])
        * half_loss
        + float(policy['deep_supervision_weights']['quarter_resolution'])
        * quarter_loss
        + float(policy['territory_node_loss_weight']) * node_loss
    )
    return total, {
        'total_loss': total,
        'main_loss': main_loss,
        'soft_dice_loss': main_parts['soft_dice'],
        'boundary_bce_loss': main_parts['boundary_bce'],
        'hard_focal_tversky_loss': main_parts['hard_focal_tversky'],
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
        'spatially_augmented_fraction': torch.as_tensor(
            spatial_fraction, device=device
        ),
    }


def epoch_loader(dataset, epoch):
    dataset.set_epoch(epoch)
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda',
        persistent_workers=False,
        drop_last=False,
    )


def update_ema(ema_model, study_model, maximum_decay, update_step):
    
    
    decay = min(
        float(maximum_decay),
        float(update_step + 1) / float(update_step + 10),
    )
    with torch.no_grad():
        for ema_parameter, parameter in zip(
            ema_model.parameters(), study_model.parameters()
        ):
            ema_parameter.mul_(decay).add_(parameter, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(
            ema_model.buffers(), study_model.buffers()
        ):
            ema_buffer.copy_(buffer)


def threshold_counts(probability, target):
    probability = np.asarray(probability, dtype=np.float32).reshape(-1)
    target = np.asarray(target, dtype=bool).reshape(-1)
    edges = np.concatenate((
        np.asarray([-np.inf], dtype=np.float32),
        THRESHOLDS.astype(np.float32),
        np.asarray([np.inf], dtype=np.float32),
    ))
    positive_histogram = np.histogram(
        probability[target], bins=edges
    )[0].astype(np.int64)
    negative_histogram = np.histogram(
        probability[~target], bins=edges
    )[0].astype(np.int64)
    true_positive = np.cumsum(positive_histogram[::-1])[::-1][1:]
    false_positive = np.cumsum(negative_histogram[::-1])[::-1][1:]
    truth = int(target.sum())
    predicted = true_positive + false_positive
    denominator = predicted + truth
    dice = np.divide(
        2.0 * true_positive,
        denominator,
        out=np.ones_like(denominator, dtype=np.float64),
        where=denominator > 0,
    )
    precision = np.divide(
        true_positive,
        predicted,
        out=np.where(truth == 0, 1.0, 0.0).astype(np.float64)
        * np.ones_like(predicted, dtype=np.float64),
        where=predicted > 0,
    )
    recall = (
        np.ones_like(true_positive, dtype=np.float64)
        if truth == 0
        else true_positive.astype(np.float64) / truth
    )
    return true_positive, predicted, truth, dice, precision, recall


@torch.no_grad()
def validate_all_subjects(study_model, rows, epoch):
    study_model.eval()
    case_rows = []
    for case_index, row in enumerate(rows, start=1):
        subject_id = str(row['subject_id'])
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
            raise RuntimeError(f'{subject_id}: validation NPZ hash changed.')
        with np.load(npz_path, allow_pickle=False) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target = np.asarray(data['target_binary'], dtype=np.uint8)
            spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        image = clinical_window_recalibration(image)
        region_map, _ = load_region_map(row, target.shape)
        graph_inputs = load_graph_inputs(row)
        probability = sliding_window_probability(
            study_model, row, image, region_map, graph_inputs
        )
        (
            intersection,
            predicted,
            truth,
            dice,
            precision,
            recall,
        ) = threshold_counts(probability, target > 0)
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        for threshold_index, threshold in enumerate(THRESHOLDS):
            predicted_voxels = int(predicted[threshold_index])
            case_rows.append({
                'epoch': epoch,
                'subject_id': subject_id,
                'lesion_volume_bin': str(
                    row.get('lesion_volume_bin', 'unknown')
                ),
                'threshold': float(threshold),
                'dice': float(dice[threshold_index]),
                'precision': float(precision[threshold_index]),
                'recall': float(recall[threshold_index]),
                'intersection': int(intersection[threshold_index]),
                'predicted_voxels': predicted_voxels,
                'target_voxels': truth,
                'predicted_volume_ml': predicted_voxels * voxel_volume_ml,
                'target_volume_ml': truth * voxel_volume_ml,
                'absolute_volume_error_ml': (
                    abs(predicted_voxels - truth) * voxel_volume_ml
                ),
            })
        if (
            case_index == 1
            or case_index % 5 == 0
            or case_index == len(rows)
        ):
            print(f'  Validated {case_index}/{len(rows)}: {subject_id}')
    case_frame = pd.DataFrame(case_rows)
    summary = (
        case_frame.groupby('threshold')
        .agg(
            mean_subject_dice=('dice', 'mean'),
            median_subject_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_error_ml=(
                'absolute_volume_error_ml', 'mean'
            ),
            total_intersection=('intersection', 'sum'),
            total_predicted_voxels=('predicted_voxels', 'sum'),
            total_target_voxels=('target_voxels', 'sum'),
        )
        .reset_index()
    )
    summary['global_dice'] = (
        2.0 * summary['total_intersection']
        / np.maximum(
            summary['total_predicted_voxels']
            + summary['total_target_voxels'],
            1,
        )
    )
    summary = summary.sort_values(
        ['mean_subject_dice', 'median_subject_dice', 'mean_recall', 'threshold'],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = summary.iloc[0].to_dict()
    selected_threshold = float(selected['threshold'])
    selected_cases = case_frame[
        np.isclose(case_frame['threshold'], selected_threshold)
    ].copy()
    lesion_bin_summary = (
        selected_cases.groupby('lesion_volume_bin', dropna=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_dice=('dice', 'mean'),
            median_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
        )
        .reset_index()
        .sort_values('lesion_volume_bin')
    )
    epoch_directory = VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_directory.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(case_frame, epoch_directory / 'validation_cases.csv')
    atomic_write_csv(summary, epoch_directory / 'threshold_summary.csv')
    atomic_write_csv(
        lesion_bin_summary,
        epoch_directory / 'lesion_bin_summary.csv',
    )
    selected['lesion_bin_mean_dice'] = {
        str(row['lesion_volume_bin']): float(row['mean_dice'])
        for row in lesion_bin_summary.to_dict('records')
    }
    atomic_write_json(
        {
            'epoch': epoch,
            'validation_model': 'ema',
            'selected_threshold': selected_threshold,
            'mean_subject_dice': float(selected['mean_subject_dice']),
            'median_subject_dice': float(selected['median_subject_dice']),
            'global_dice': float(selected['global_dice']),
            'mean_precision': float(selected['mean_precision']),
            'mean_recall': float(selected['mean_recall']),
            'lesion_bin_mean_dice': selected['lesion_bin_mean_dice'],
            'test_targets_read': False,
        },
        epoch_directory / 'validation_summary.json',
    )
    return selected





training_dataset = Trial8V3PatchDataset(train_rows, SEED + 31)
if len(training_dataset) != SAMPLES_PER_EPOCH:
    raise RuntimeError('V3 dataset epoch length changed.')

updates_per_epoch = math.ceil(
    math.ceil(SAMPLES_PER_EPOCH / BATCH_SIZE) / ACCUMULATION_STEPS
)
total_update_steps = MAX_EPOCHS * updates_per_epoch
warmup_steps = 3 * updates_per_epoch


def learning_rate_multiplier(step):
    if step < warmup_steps:
        return float(step + 1) / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(
        total_update_steps - warmup_steps, 1
    )
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


set_determinism(SEED + 37)
training_model = VascuFateNet(
    schema=graph_schema,
    image_channels=15,
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
if parameter_count(training_model) != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Fresh v3 model parameter count changed.')

ema_model = copy.deepcopy(training_model).to(device).eval()
ema_model.requires_grad_(False)
if parameter_count(ema_model) != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('EMA shadow parameter count changed.')
if parameter_count(ema_model, True) != 0:
    raise RuntimeError('EMA shadow unexpectedly has trainable parameters.')

training_optimizer = torch.optim.AdamW(
    training_model.parameters(),
    lr=BASE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
training_scheduler = torch.optim.lr_scheduler.LambdaLR(
    training_optimizer, lr_lambda=learning_rate_multiplier
)
training_amp_scaler = torch.amp.GradScaler(
    device.type, enabled=AMP_SCALER_ENABLED
)

start_epoch = 1
global_update_step = 0
best_mean_dice = -1.0
best_epoch = 0
best_threshold = 0.50
validations_without_improvement = 0
history_rows = []

if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'training_version_matches': checkpoint.get('training_version')
        == TRAINING_VERSION,
        'v3_gate_hash_matches': checkpoint.get('v3_gate_sha256')
        == v3_gate_sha256,
        'manifest_hash_matches': checkpoint.get(
            'verified_manifest_sha256'
        ) == verified_manifest_sha256,
        'scaler_hash_matches': checkpoint.get('scaler_sha256')
        == EXPECTED_SCALER_SHA256,
        'schema_hash_matches': checkpoint.get('graph_schema_sha256')
        == safe_schema_sha256,
        'parameter_count_matches': checkpoint.get('total_parameters')
        == EXPECTED_TOTAL_PARAMETERS,
        'test_targets_not_read': checkpoint.get('test_targets_read')
        is False,
        'ema_state_present': 'ema_model_state_dict' in checkpoint,
    }
    if not all(checkpoint_checks.values()):
        failed = [
            name for name, passed in checkpoint_checks.items() if not passed
        ]
        raise RuntimeError('V3 checkpoint mismatch: ' + ', '.join(failed))
    training_model.load_state_dict(checkpoint['model_state_dict'])
    ema_model.load_state_dict(checkpoint['ema_model_state_dict'])
    training_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    training_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if AMP_SCALER_ENABLED:
        training_amp_scaler.load_state_dict(
            checkpoint['amp_scaler_state_dict']
        )
    restore_rng_state(checkpoint['rng_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    global_update_step = int(checkpoint['global_update_step'])
    best_mean_dice = float(checkpoint['best_mean_dice'])
    best_epoch = int(checkpoint['best_epoch'])
    best_threshold = float(checkpoint['best_threshold'])
    validations_without_improvement = int(
        checkpoint['validations_without_improvement']
    )
    history_rows = list(checkpoint.get('history_rows', []))
    print(
        f'Resuming Trial8 v3 after epoch {start_epoch - 1}; '
        f'best EMA Dice={best_mean_dice:.4f} at epoch {best_epoch}.'
    )
else:
    print('Starting Trial8 v3 from a fresh deterministic initialization.')

training_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_version': TRAINING_VERSION,
    'code_revision': CODE_REVISION,
    'training_policy': policy,
    'v3_gate_path': str(V3_GATE_PATH),
    'v3_gate_sha256': v3_gate_sha256,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_sha256': EXPECTED_SCALER_SHA256,
    'graph_schema_sha256': safe_schema_sha256,
    'architecture_sha256': architecture_sha256,
    'total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
    'ema_shadow_parameters': EXPECTED_TOTAL_PARAMETERS,
    'ema_shadow_trainable_parameters': 0,
    'train_subjects': 102,
    'validation_subjects': 23,
    'test_subjects_accessed': 0,
    'mixed_precision_dtype': MIXED_PRECISION_NAME,
    'gradient_scaler_enabled': AMP_SCALER_ENABLED,
    'loss_reduction_dtype': 'float32',
    'focal_tversky_error_floor': 1e-6,
}
if TRAINING_CONFIG_PATH.exists():
    existing = json.loads(
        TRAINING_CONFIG_PATH.read_text(encoding='utf-8')
    )
    immutable = [
        'training_version',
        'training_policy',
        'v3_gate_sha256',
        'verified_manifest_sha256',
        'scaler_sha256',
        'graph_schema_sha256',
        'architecture_sha256',
        'total_parameters',
    ]
    if any(existing.get(key) != training_config.get(key) for key in immutable):
        raise RuntimeError('Existing v3 training configuration changed.')
    existing['code_revision'] = CODE_REVISION
    existing['mixed_precision_dtype'] = MIXED_PRECISION_NAME
    existing['gradient_scaler_enabled'] = AMP_SCALER_ENABLED
    existing['loss_reduction_dtype'] = 'float32'
    existing['focal_tversky_error_floor'] = 1e-6
    atomic_write_json(existing, TRAINING_CONFIG_PATH)
else:
    atomic_write_json(training_config, TRAINING_CONFIG_PATH)

print('\n' + '=' * 80)
print('TRIAL8 V3 TOPOLOGY-CURRICULUM TRAINING START')
print('=' * 80)
print(f'Device:                             {device}')
if device.type == 'cuda':
    print(f'GPU:                                {torch.cuda.get_device_name(0)}')
print('Train / validation / test access:   102 / 23 / 0')
print(f'Patch size:                         {PATCH_SIZE}')
print(f'Samples per epoch:                  {SAMPLES_PER_EPOCH}')
print('Positive/topology/brain samples:    102 / 51 / 51')
print(f'Batch / accumulation / effective:   {BATCH_SIZE} / {ACCUMULATION_STEPS} / {BATCH_SIZE * ACCUMULATION_STEPS}')
print(f'Updates per epoch:                  {updates_per_epoch}')
print(f'Maximum epochs:                     {MAX_EPOCHS}')
print(f'Early-stopping patience:            {PATIENCE_VALIDATIONS} epochs')
print(f'Learning rate / weight decay:       {BASE_LEARNING_RATE:.1e} / {WEIGHT_DECAY:.1e}')
print(f'EMA decay:                          {EMA_DECAY:.3f}')
print(f'Mixed precision:                    {MIXED_PRECISION_NAME}')
print('Loss reductions:                    float32')
print('Focal-Tversky derivative floor:     1e-6')
print(f'Total/trainable parameters:         {EXPECTED_TOTAL_PARAMETERS:,} / {EXPECTED_TOTAL_PARAMETERS:,}')
print(f'EMA shadow/trainable parameters:    {EXPECTED_TOTAL_PARAMETERS:,} / 0')
print('Clinical windows:                   CTA/CBF/MTT/Tmax recalibrated')
print('Novel curriculum:                   vascular-topology-aware negatives')
print(f'Checkpoint directory:               {TRAINING_ROOT}')





training_already_complete = (
    start_epoch > MAX_EPOCHS
    or validations_without_improvement >= PATIENCE_VALIDATIONS
)
stop_reason = (
    'already_complete_on_resume'
    if training_already_complete
    else 'maximum_epochs_reached'
)
if training_already_complete:
    print('The v3 checkpoint already satisfies a terminal stop condition.')

for epoch in range(
    start_epoch,
    start_epoch if training_already_complete else MAX_EPOCHS + 1,
):
    epoch_start = datetime.now(timezone.utc)
    loader = epoch_loader(training_dataset, epoch)
    training_model.train()
    training_optimizer.zero_grad(set_to_none=True)
    running = {
        'total_loss': 0.0,
        'main_loss': 0.0,
        'soft_dice_loss': 0.0,
        'boundary_bce_loss': 0.0,
        'hard_focal_tversky_loss': 0.0,
        'half_loss': 0.0,
        'quarter_loss': 0.0,
        'node_loss': 0.0,
        'atlas_retained_fraction': 0.0,
        'spatially_augmented_fraction': 0.0,
    }
    processed_batches = 0
    processed_samples = 0
    positive_samples_seen = 0
    topology_samples_seen = 0
    brain_negative_samples_seen = 0
    gradient_norms = []
    amp_skipped_updates = 0
    accumulation_subjects = []
    accumulation_sample_types = []
    for batch_index, cpu_batch in enumerate(loader, start=1):
        batch_samples = int(cpu_batch['target_binary'].shape[0])
        processed_samples += batch_samples
        positive_samples_seen += sum(
            str(value) == 'positive' for value in cpu_batch['sample_type']
        )
        topology_samples_seen += sum(
            str(value) == 'topology_negative'
            for value in cpu_batch['sample_type']
        )
        brain_negative_samples_seen += sum(
            str(value) == 'brain_negative'
            for value in cpu_batch['sample_type']
        )
        accumulation_subjects.extend(
            str(value) for value in cpu_batch['subject_id']
        )
        accumulation_sample_types.extend(
            str(value) for value in cpu_batch['sample_type']
        )
        batch = to_device(cpu_batch)
        with torch.autocast(
            device_type=device.type,
            dtype=MIXED_PRECISION_DTYPE,
            enabled=device.type == 'cuda',
        ):
            total_loss, parts = compute_training_loss(
                training_model, batch
            )
            accumulation_group_start = (
                (batch_index - 1) // ACCUMULATION_STEPS
            ) * ACCUMULATION_STEPS
            accumulation_group_size = min(
                ACCUMULATION_STEPS,
                len(loader) - accumulation_group_start,
            )
            scaled_loss = total_loss / accumulation_group_size
        training_amp_scaler.scale(scaled_loss).backward()
        for name in running:
            running[name] += float(parts[name].detach().cpu())
        processed_batches += 1
        should_update = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(loader)
        )
        if should_update:
            training_amp_scaler.unscale_(training_optimizer)
            gradient_norm = float(
                torch.nn.utils.clip_grad_norm_(
                    training_model.parameters(), GRADIENT_CLIP_NORM
                ).item()
            )
            if not math.isfinite(gradient_norm):
                nonfinite_parameters = [
                    name
                    for name, parameter in training_model.named_parameters()
                    if parameter.grad is not None
                    and not bool(torch.isfinite(parameter.grad).all().item())
                ]
                training_optimizer.zero_grad(set_to_none=True)
                raise RuntimeError(
                    'Non-finite bfloat16/float32 gradient at '
                    f'epoch {epoch}, batch {batch_index}; '
                    f'subjects={accumulation_subjects}; '
                    f'sample_types={accumulation_sample_types}; '
                    f'parameters={nonfinite_parameters[:20]}.'
                )
            gradient_norms.append(gradient_norm)
            training_amp_scaler.step(training_optimizer)
            training_amp_scaler.update()
            training_optimizer.zero_grad(set_to_none=True)
            training_scheduler.step()
            global_update_step += 1
            update_ema(
                ema_model,
                training_model,
                EMA_DECAY,
                global_update_step,
            )
            accumulation_subjects.clear()
            accumulation_sample_types.clear()

    if not gradient_norms:
        raise RuntimeError(
            f'Every optimizer update was skipped at epoch {epoch}.'
        )

    epoch_means = {
        name: value / max(processed_batches, 1)
        for name, value in running.items()
    }
    if not all(math.isfinite(value) for value in epoch_means.values()):
        raise RuntimeError(f'Non-finite v3 loss at epoch {epoch}.')
    curriculum_counts_ok = (
        positive_samples_seen == POSITIVE_SAMPLES
        and topology_samples_seen == TOPOLOGY_NEGATIVE_SAMPLES
        and brain_negative_samples_seen == BRAIN_NEGATIVE_SAMPLES
        and processed_samples == SAMPLES_PER_EPOCH
    )
    if not curriculum_counts_ok:
        raise RuntimeError(f'V3 epoch {epoch} curriculum counts changed.')

    print(f'Running full EMA validation at epoch {epoch}...')
    validation_result = validate_all_subjects(
        ema_model, validation_rows, epoch
    )
    current_dice = float(validation_result['mean_subject_dice'])
    current_threshold = float(validation_result['threshold'])
    improved = current_dice > best_mean_dice + IMPROVEMENT_EPSILON
    if improved:
        best_mean_dice = current_dice
        best_epoch = epoch
        best_threshold = current_threshold
        validations_without_improvement = 0
    else:
        validations_without_improvement += 1

    epoch_seconds = (
        datetime.now(timezone.utc) - epoch_start
    ).total_seconds()
    history_row = {
        'epoch': epoch,
        'global_update_step': global_update_step,
        'learning_rate': float(training_optimizer.param_groups[0]['lr']),
        **epoch_means,
        'mean_gradient_norm': float(np.mean(gradient_norms)),
        'maximum_gradient_norm': float(np.max(gradient_norms)),
        'amp_skipped_updates': 0,
        'mixed_precision_dtype': MIXED_PRECISION_NAME,
        'positive_patch_fraction': (
            positive_samples_seen / processed_samples
        ),
        'topology_negative_fraction': (
            topology_samples_seen / processed_samples
        ),
        'brain_negative_fraction': (
            brain_negative_samples_seen / processed_samples
        ),
        'validation_model': 'ema',
        'validation_threshold': current_threshold,
        'validation_mean_subject_dice': current_dice,
        'validation_median_subject_dice': float(
            validation_result['median_subject_dice']
        ),
        'validation_global_dice': float(validation_result['global_dice']),
        'validation_mean_precision': float(
            validation_result['mean_precision']
        ),
        'validation_mean_recall': float(
            validation_result['mean_recall']
        ),
        'validation_mean_absolute_volume_error_ml': float(
            validation_result['mean_absolute_volume_error_ml']
        ),
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'improved': improved,
        'epoch_seconds': epoch_seconds,
        'test_targets_read': False,
    }
    for lesion_bin, value in validation_result[
        'lesion_bin_mean_dice'
    ].items():
        history_row[f'validation_mean_dice_{lesion_bin}'] = float(value)
    history_rows.append(to_builtin(history_row))
    atomic_write_csv(pd.DataFrame(history_rows), HISTORY_PATH)

    checkpoint_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'epoch': epoch,
        'global_update_step': global_update_step,
        'model_state_dict': training_model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': training_optimizer.state_dict(),
        'scheduler_state_dict': training_scheduler.state_dict(),
        'amp_scaler_state_dict': training_amp_scaler.state_dict(),
        'rng_state': capture_rng_state(),
        'best_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'history_rows': history_rows,
        'code_revision': CODE_REVISION,
        'amp_skipped_updates_this_epoch': 0,
        'mixed_precision_dtype': MIXED_PRECISION_NAME,
        'v3_gate_sha256': v3_gate_sha256,
        'verified_manifest_sha256': verified_manifest_sha256,
        'scaler_sha256': EXPECTED_SCALER_SHA256,
        'graph_schema_sha256': safe_schema_sha256,
        'architecture_sha256': architecture_sha256,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'test_targets_read': False,
    }
    atomic_torch_save(checkpoint_record, LAST_CHECKPOINT_PATH)
    if improved:
        atomic_torch_save(checkpoint_record, BEST_CHECKPOINT_PATH)

    print(
        f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
        f'loss={epoch_means["total_loss"]:.4f}, '
        f'main={epoch_means["main_loss"]:.4f}, '
        f'node={epoch_means["node_loss"]:.4f}, '
        f'lr={history_row["learning_rate"]:.2e}, '
        f'grad={history_row["mean_gradient_norm"]:.3f}, '
        'amp-skips=0, '
        f'positive={history_row["positive_patch_fraction"]:.3f}, '
        f'topology-neg={history_row["topology_negative_fraction"]:.3f}, '
        f'brain-neg={history_row["brain_negative_fraction"]:.3f}, '
        f'time={epoch_seconds / 60.0:.1f} min'
    )
    print(
        f'  EMA validation: mean Dice={current_dice:.4f}, '
        f'median Dice={validation_result["median_subject_dice"]:.4f}, '
        f'global Dice={validation_result["global_dice"]:.4f}, '
        f'precision={validation_result["mean_precision"]:.4f}, '
        f'recall={validation_result["mean_recall"]:.4f}, '
        f'threshold={current_threshold:.3f}, '
        f'best={best_mean_dice:.4f} at epoch {best_epoch}'
    )
    print(
        '  Mean Dice by lesion bin: '
        + ', '.join(
            f'{name}={value:.4f}'
            for name, value in validation_result[
                'lesion_bin_mean_dice'
            ].items()
        )
    )

    atomic_write_json(
        {
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'training_version': TRAINING_VERSION,
            'status': 'running',
            'last_completed_epoch': epoch,
            'global_update_step': global_update_step,
            'best_mean_subject_dice': best_mean_dice,
            'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'validations_without_improvement': (
                validations_without_improvement
            ),
            'total_parameters': EXPECTED_TOTAL_PARAMETERS,
            'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
            'ema_shadow_trainable_parameters': 0,
            'test_targets_read': False,
            'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
            'best_checkpoint_path': (
                str(BEST_CHECKPOINT_PATH)
                if BEST_CHECKPOINT_PATH.exists()
                else ''
            ),
        },
        TRAINING_STATE_PATH,
    )

    if validations_without_improvement >= PATIENCE_VALIDATIONS:
        stop_reason = 'early_stopping'
        print(
            f'Early stopping after {validations_without_improvement} '
            'full EMA validations without improvement.'
        )
        break

completed_epoch = history_rows[-1]['epoch'] if history_rows else start_epoch - 1
atomic_write_json(
    {
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'status': 'complete',
        'stop_reason': stop_reason,
        'last_completed_epoch': completed_epoch,
        'global_update_step': global_update_step,
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
        'ema_shadow_trainable_parameters': 0,
        'train_subjects': 102,
        'validation_subjects': 23,
        'test_targets_read': False,
        'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
        'best_checkpoint_path': str(BEST_CHECKPOINT_PATH),
        'history_path': str(HISTORY_PATH),
    },
    TRAINING_STATE_PATH,
)

print('\n' + '=' * 80)
print('TRIAL8 V3 TOPOLOGY-CURRICULUM TRAINING COMPLETE')
print('=' * 80)
print(f'Stop reason:                        {stop_reason}')
print(f'Last completed epoch:               {completed_epoch}')
print(f'Best EMA validation mean Dice:      {best_mean_dice:.4f}')
print(f'Best epoch:                         {best_epoch}')
print(f'Frozen validation threshold:        {best_threshold:.3f}')
print(f'Total/trainable parameters:         {EXPECTED_TOTAL_PARAMETERS:,} / {EXPECTED_TOTAL_PARAMETERS:,}')
print(f'EMA shadow/trainable parameters:    {EXPECTED_TOTAL_PARAMETERS:,} / 0')
print('Test targets read:                  False')
print('Validation cadence:                 every epoch')
print('Novel curriculum:                   vascular-topology-aware negatives')
print('Scientific core changed:            False')
print(f'Best checkpoint:                    {BEST_CHECKPOINT_PATH}')
print(f'Last checkpoint:                    {LAST_CHECKPOINT_PATH}')
print(f'Training history:                   {HISTORY_PATH}')
print(f'Training state:                     {TRAINING_STATE_PATH}')


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
V4_PREFLIGHT_VERSION = 'v4_scale_balanced_ohem_curriculum'
CODE_REVISION = 'r1_subject_covered_staged_curriculum'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_PARAMETERS = 15_052_628
EXPECTED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_VERIFICATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / VERIFICATION_VERSION
)
GRAPH_VERIFICATION_GATE_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_verification_gate.json'
)
GRAPH_SCHEMA_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)
MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / 'v1_train_only_scaler_ragatv2_film'
)
MODEL_PREFLIGHT_GATE_PATH = (
    MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'
)
SCALER_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
THRESHOLD_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT
    / 'checkpoints'
    / MODEL_VERSION
    / 'v1_dense_threshold_v1_v2_probability_ensemble'
)
THRESHOLD_DIAGNOSTIC_GATE_PATH = (
    THRESHOLD_DIAGNOSTIC_ROOT / 'diagnostic_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / V4_PREFLIGHT_VERSION
)
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_target_audit.csv'
SAMPLING_AUDIT_PATH = OUTPUT_ROOT / 'topology_sampling_audit.csv'
WINDOW_AUDIT_PATH = OUTPUT_ROOT / 'clinical_window_audit.csv'
POLICY_PATH = OUTPUT_ROOT / 'v4_training_policy.json'
GATE_PATH = OUTPUT_ROOT / 'v4_sampler_loss_preflight_gate.json'

SEED = 20260920
PATCH_SIZE = (112, 112, 48)
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
POSITIVE_SAMPLES = 144
TOPOLOGY_NEGATIVE_SAMPLES = 30
BRAIN_NEGATIVE_SAMPLES = 30
EPOCH_SAMPLES = (
    POSITIVE_SAMPLES
    + TOPOLOGY_NEGATIVE_SAMPLES
    + BRAIN_NEGATIVE_SAMPLES
)
MAX_CENTER_CANDIDATES = 512
CASE_WEIGHT_MIN = 0.75
CASE_WEIGHT_MAX = 3.00


def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    return str(value).strip().lower() in {'true', '1', 'yes'}


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    return value


def atomic_write_json(record, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


required_paths = [
    VERIFIED_MANIFEST_PATH,
    GRAPH_VERIFICATION_GATE_PATH,
    GRAPH_SCHEMA_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
    SCALER_PATH,
    THRESHOLD_DIAGNOSTIC_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 v3 preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
scaler_sha256 = file_sha256(SCALER_PATH)
if manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError('Verified manifest hash changed.')
if schema_sha256 != EXPECTED_SCHEMA_SHA256:
    raise RuntimeError('Verified graph schema hash changed.')
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Train-only scaler hash changed.')

graph_gate = json.loads(
    GRAPH_VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
diagnostic_gate = json.loads(
    THRESHOLD_DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8')
)

upstream_checks = {
    'graph_verification_passed': graph_gate.get(
        'verification_passed'
    ) is True,
    'graph_training_authorized': graph_gate.get(
        'model_training_authorized'
    ) is True,
    'model_preflight_passed': model_gate.get(
        'preflight_passed'
    ) is True,
    'model_training_authorized': model_gate.get(
        'model_training_authorized'
    ) is True,
    'model_parameter_count_unchanged': model_gate.get(
        'total_parameters'
    ) == EXPECTED_PARAMETERS,
    'threshold_diagnostic_passed': diagnostic_gate.get(
        'diagnostic_passed'
    ) is True,
    'threshold_diagnostic_target_safe': diagnostic_gate.get(
        'test_targets_read'
    ) is False,
    'threshold_diagnostic_architecture_unchanged': diagnostic_gate.get(
        'scientific_core_changed'
    ) is False,
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream v3 gates failed: ' + ', '.join(failed))

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'lesion_volume_bin',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(
        'Verified manifest is missing columns: ' + ', '.join(missing_columns)
    )
manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Manifest checks failed: ' + ', '.join(failed))

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}
territory_edge_index = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)

print('Trial8 v4 upstream gates passed.')
print(f'Verified subjects:                  {len(manifest_df)}')
print(f'Split counts:                       {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Frozen manifest SHA-256:            {manifest_sha256}')
print(f'Pickle-free schema SHA-256:         {schema_sha256}')
print('Target arrays authorized:           train only')
print('Test targets read:                  False')
print('Study trainable parameters:         0 (preflight only)')





train_df = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .reset_index(drop=True)
)
target_rows = []
for row in train_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'crop_mask',
            'target_spacing',
            'image',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing arrays: {missing}')
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask']) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        image_shape = tuple(int(value) for value in data['image'].shape)
    if target.shape != soft.shape or target.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if image_shape[0] != 15 or image_shape[1:] != target.shape:
        raise RuntimeError(f'{subject_id}: image shape is inconsistent.')
    if not np.isfinite(soft).all():
        raise RuntimeError(f'{subject_id}: soft target is non-finite.')
    target_voxels = int(target.sum())
    volume_ml = float(target_voxels * np.prod(spacing) / 1000.0)
    target_rows.append({
        'subject_id': subject_id,
        'lesion_volume_bin': str(row['lesion_volume_bin']),
        'target_voxels': target_voxels,
        'lesion_volume_ml': volume_ml,
        'target_inside_crop_fraction': (
            1.0
            if target_voxels == 0
            else float((target.astype(bool) & crop_mask).sum() / target_voxels)
        ),
        'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
        'npz_path': str(npz_path),
    })

target_audit_df = pd.DataFrame(target_rows)
nonempty_volumes = target_audit_df.loc[
    target_audit_df['lesion_volume_ml'] > 0,
    'lesion_volume_ml',
].to_numpy(dtype=np.float64)
median_nonempty_volume_ml = float(np.median(nonempty_volumes))


def lesion_case_weight(volume_ml):
    if volume_ml <= 0:
        return 1.0
    return float(np.clip(
        math.sqrt(median_nonempty_volume_ml / volume_ml),
        CASE_WEIGHT_MIN,
        CASE_WEIGHT_MAX,
    ))


target_audit_df['positive_case_loss_weight'] = target_audit_df[
    'lesion_volume_ml'
].map(lesion_case_weight)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)

target_by_subject = target_audit_df.set_index('subject_id').to_dict('index')
manifest_by_subject = {
    str(row['subject_id']): row for row in train_df.to_dict('records')
}






def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def integral_volume(binary):
    return np.pad(
        np.asarray(binary, dtype=np.int64),
        ((1, 0), (1, 0), (1, 0)),
        mode='constant',
    ).cumsum(0).cumsum(1).cumsum(2)


def box_sum(integral, slices):
    x0, x1 = slices[0].start, slices[0].stop
    y0, y1 = slices[1].start, slices[1].stop
    z0, z1 = slices[2].start, slices[2].stop
    return int(
        integral[x1, y1, z1]
        - integral[x0, y1, z1]
        - integral[x1, y0, z1]
        - integral[x1, y1, z0]
        + integral[x0, y0, z1]
        + integral[x0, y1, z0]
        + integral[x1, y0, z0]
        - integral[x0, y0, z0]
    )


def sampled_coordinates(mask, rng, maximum=MAX_CENTER_CANDIDATES):
    flat = np.flatnonzero(np.asarray(mask, dtype=bool))
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > maximum:
        flat = rng.choice(flat, size=maximum, replace=False)
    coordinates = np.column_stack(
        np.unravel_index(flat, mask.shape)
    )
    return coordinates.astype(np.int64)


def minimum_overlap_center(target, candidate_mask, rng):
    coordinates = sampled_coordinates(candidate_mask, rng)
    if len(coordinates) == 0:
        return None, None
    integral = integral_volume(target > 0)
    counts = np.asarray([
        box_sum(
            integral,
            patch_slices_from_center(target.shape, center, PATCH_SIZE),
        )
        for center in coordinates
    ], dtype=np.int64)
    minimum = int(counts.min())
    best = np.flatnonzero(counts == minimum)
    selected = int(best[int(rng.integers(0, len(best)))])
    return coordinates[selected], minimum


def load_train_case(subject_id, include_image=False):
    row = manifest_by_subject[subject_id]
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
        image = (
            np.asarray(data['image'], dtype=np.float32)
            if include_image
            else None
        )
    region_map = np.zeros(target.shape, dtype=np.int16)
    if as_bool(row['trial8_atlas_enabled']):
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region_map = np.asanyarray(atlas_image.dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int16)
        if region_map.shape != target.shape:
            raise RuntimeError(f'{subject_id}: atlas shape mismatch.')
    return target, crop_mask, region_map, image


def topology_candidate_labels(target, region_map):
    lesion_labels = set(
        int(value)
        for value in np.unique(region_map[target > 0])
        if 1 <= int(value) <= 30
    )
    if not lesion_labels:
        return [], {}, lesion_labels
    candidates_by_priority = {2: [], 1: [], 3: []}
    relation_by_label = {}
    relation_priority = {2: 0, 1: 1, 3: 2}
    for edge_position in range(territory_edge_index.shape[1]):
        source = int(territory_edge_index[0, edge_position]) + 1
        destination = int(territory_edge_index[1, edge_position]) + 1
        relation = int(territory_edge_type[edge_position])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in candidates_by_priority
        ):
            candidates_by_priority[relation].append(destination)
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    ordered = []
    for relation in [2, 1, 3]:
        ordered.extend(sorted(set(candidates_by_priority[relation])))
    return list(dict.fromkeys(ordered)), relation_by_label, lesion_labels


RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def select_positive(subject_id, rng):
    target, crop_mask, region_map, _ = load_train_case(subject_id)
    positive = np.argwhere(target > 0)
    if len(positive) == 0:
        raise RuntimeError(f'{subject_id}: positive schedule selected empty target.')
    center = positive[int(rng.integers(0, len(positive)))].astype(np.int64)
    jitter = np.asarray([
        int(rng.integers(-12, 13)),
        int(rng.integers(-12, 13)),
        int(rng.integers(-4, 5)),
    ])
    center = np.clip(center + jitter, 0, np.asarray(target.shape) - 1)
    slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
    count = int(target[slices].sum())
    return subject_id, center, count, 'lesion_jitter', '', 0


def select_topology_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        target, crop_mask, region_map, _ = load_train_case(subject_id)
        labels, relation_by_label, _ = topology_candidate_labels(
            target, region_map
        )
        if not labels:
            continue
        for relation in [2, 1, 3]:
            relation_labels = [
                label for label in labels
                if relation_by_label.get(label) == relation
            ]
            if not relation_labels:
                continue
            candidate_mask = (
                np.isin(region_map, relation_labels) & crop_mask
            )
            center, count = minimum_overlap_center(
                target, candidate_mask, rng
            )
            if center is not None and count == 0:
                center_label = int(region_map[tuple(center)])
                return (
                    subject_id,
                    center,
                    count,
                    'topology_hard_negative',
                    RELATION_NAME[relation],
                    center_label,
                )
    raise RuntimeError(
        'Unable to construct a lesion-free topology hard negative.'
    )


def select_brain_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        target, crop_mask, region_map, _ = load_train_case(subject_id)
        center, count = minimum_overlap_center(target, crop_mask, rng)
        if center is not None and count == 0:
            return (
                subject_id,
                center,
                count,
                'brain_hard_negative',
                '',
                int(region_map[tuple(center)]),
            )
    raise RuntimeError('Unable to construct a lesion-free brain negative.')





rng = np.random.default_rng(SEED)
subjects_by_bin = {
    lesion_bin: target_audit_df[
        (target_audit_df['lesion_volume_bin'] == lesion_bin)
        & (target_audit_df['target_voxels'] > 0)
    ]['subject_id'].astype(str).tolist()
    for lesion_bin in POSITIVE_BINS
}
if any(len(subjects_by_bin[name]) == 0 for name in POSITIVE_BINS):
    raise RuntimeError('A required positive lesion-volume bin is empty.')

topology_pool = target_audit_df[
    (target_audit_df['target_voxels'] > 0)
    & target_audit_df['atlas_enabled']
]['subject_id'].astype(str).tolist()
brain_pool = target_audit_df.sort_values(
    ['target_voxels', 'subject_id']
)['subject_id'].astype(str).tolist()

sampling_rows = []
nonempty_subjects = sorted(
    target_audit_df.loc[
        target_audit_df['target_voxels'] > 0, 'subject_id'
    ].astype(str).tolist()
)
positive_subject_schedule = list(nonempty_subjects)
extra_positive_subjects = []
extra_bins = ['tiny', 'small']
extra_index = 0
while len(positive_subject_schedule) < POSITIVE_SAMPLES:
    lesion_bin = extra_bins[extra_index % len(extra_bins)]
    pool = subjects_by_bin[lesion_bin]
    extra_subject = pool[int(rng.integers(0, len(pool)))]
    positive_subject_schedule.append(extra_subject)
    extra_positive_subjects.append(extra_subject)
    extra_index += 1
rng.shuffle(positive_subject_schedule)

for sample_index, subject_id in enumerate(positive_subject_schedule):
    selected = select_positive(subject_id, rng)
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': sample_index,
        'sample_type': 'positive',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': audit['positive_case_loss_weight'],
    })

for local_index in range(TOPOLOGY_NEGATIVE_SAMPLES):
    selected = select_topology_negative(
        int(rng.integers(0, len(topology_pool))), topology_pool, rng
    )
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': POSITIVE_SAMPLES + local_index,
        'sample_type': 'topology_negative',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': 1.0,
    })

for local_index in range(BRAIN_NEGATIVE_SAMPLES):
    selected = select_brain_negative(
        int(rng.integers(0, len(brain_pool))), brain_pool, rng
    )
    audit = target_by_subject[selected[0]]
    sampling_rows.append({
        'sample_index': (
            POSITIVE_SAMPLES + TOPOLOGY_NEGATIVE_SAMPLES + local_index
        ),
        'sample_type': 'brain_negative',
        'subject_id': selected[0],
        'lesion_volume_bin': audit['lesion_volume_bin'],
        'center_source': selected[3],
        'topology_relation': selected[4],
        'center_territory_label': selected[5],
        'patch_target_voxels': selected[2],
        'case_loss_weight': 1.0,
    })

sampling_audit_df = pd.DataFrame(sampling_rows)
atomic_write_csv(sampling_audit_df, SAMPLING_AUDIT_PATH)






def apply_clinical_window_recalibration(image):
    output = np.asarray(image, dtype=np.float32).copy()
    
    
    
    output[2] = np.clip((output[2] * 1100.0 - 100.0) / 90.0, 0.0, 1.0)
    output[3] = np.clip((output[3] * 150.0) / 35.0, 0.0, 1.0)
    
    output[5] = np.clip((output[5] * 30.0) / 20.0, 0.0, 1.0)
    output[6] = np.clip((output[6] * 20.0) / 7.0, 0.0, 1.0)
    return output


window_rows = []
window_subjects = train_df.sort_values('subject_id').iloc[
    np.linspace(0, len(train_df) - 1, 12, dtype=int)
]
for row in window_subjects.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        image = np.asarray(data['image'], dtype=np.float32)
    recalibrated = apply_clinical_window_recalibration(image)
    for channel_index, channel_name in [
        (2, 'cta_0_90'),
        (3, 'cbf_0_35'),
        (5, 'mtt_0_20'),
        (6, 'tmax_0_7'),
    ]:
        before = image[channel_index]
        after = recalibrated[channel_index]
        window_rows.append({
            'subject_id': subject_id,
            'channel_index': channel_index,
            'channel_name': channel_name,
            'minimum': float(after.min()),
            'maximum': float(after.max()),
            'mean': float(after.mean(dtype=np.float64)),
            'changed_fraction': float(
                (~np.isclose(before, after, atol=1e-7)).mean()
            ),
            'all_finite': bool(np.isfinite(after).all()),
        })

window_audit_df = pd.DataFrame(window_rows)
atomic_write_csv(window_audit_df, WINDOW_AUDIT_PATH)





positive_counts = (
    sampling_audit_df[
        sampling_audit_df['sample_type'] == 'positive'
    ]['lesion_volume_bin'].value_counts()
)
sampling_checks = {
    'exactly_102_train_targets_audited': len(target_audit_df) == 102,
    'target_audit_contains_no_validation_or_test_rows': True,
    
    
    
    
    
    'target_crop_support_fractions_are_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'all_five_positive_bins_present': set(positive_counts.index)
    == set(POSITIVE_BINS),
    'every_nonempty_train_subject_has_positive_coverage': set(
        nonempty_subjects
    ).issubset(set(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'positive', 'subject_id'
        ].astype(str)
    )),
    'extra_positive_samples_are_tiny_or_small': set(
        extra_positive_subjects
    ).issubset(set(subjects_by_bin['tiny'] + subjects_by_bin['small'])),
    'exactly_144_positive_patches': int(
        (sampling_audit_df['sample_type'] == 'positive').sum()
    ) == POSITIVE_SAMPLES,
    'stage1_positive_fraction_is_expected': math.isclose(
        float((sampling_audit_df['patch_target_voxels'] > 0).mean()),
        POSITIVE_SAMPLES / EPOCH_SAMPLES,
    ),
    'all_positive_patches_contain_target': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'positive',
            'patch_target_voxels',
        ].gt(0).all()
    ),
    'exactly_30_topology_negatives': int(
        (sampling_audit_df['sample_type'] == 'topology_negative').sum()
    ) == TOPOLOGY_NEGATIVE_SAMPLES,
    'all_topology_negatives_are_lesion_free': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'patch_target_voxels',
        ].eq(0).all()
    ),
    'all_topology_negatives_have_graph_relation': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'topology_relation',
        ].astype(str).str.len().gt(0).all()
    ),
    'exactly_30_brain_negatives': int(
        (sampling_audit_df['sample_type'] == 'brain_negative').sum()
    ) == BRAIN_NEGATIVE_SAMPLES,
    'all_brain_negatives_are_lesion_free': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'brain_negative',
            'patch_target_voxels',
        ].eq(0).all()
    ),
    'case_weights_are_capped': bool(
        target_audit_df['positive_case_loss_weight'].between(
            CASE_WEIGHT_MIN, CASE_WEIGHT_MAX
        ).all()
    ),
}
window_checks = {
    'four_recalibrated_channels_audited': (
        window_audit_df['channel_name'].nunique() == 4
    ),
    'recalibrated_channels_are_finite': bool(
        window_audit_df['all_finite'].all()
    ),
    'recalibrated_channels_are_in_unit_interval': bool(
        window_audit_df['minimum'].ge(0.0).all()
        and window_audit_df['maximum'].le(1.0).all()
    ),
    'recalibration_materially_changes_each_channel': bool(
        window_audit_df.groupby('channel_name')['changed_fraction']
        .mean().gt(0.001).all()
    ),
}

training_policy = {
    'policy_version': V4_PREFLIGHT_VERSION,
    'scientific_core': (
        '15-channel residual 3D U-Net plus heterogeneous relation-aware '
        'GATv2 and bounded zero-init multiscale FiLM'
    ),
    'scientific_core_changed': False,
    'novel_training_component': (
        'subject-covered lesion-scale curriculum with vascular-topology '
        'hard negatives and hard-background-mined supervision'
    ),
    'patch_size': list(PATCH_SIZE),
    'samples_per_epoch': EPOCH_SAMPLES,
    'positive_samples_per_epoch': POSITIVE_SAMPLES,
    'topology_negative_samples_per_epoch': TOPOLOGY_NEGATIVE_SAMPLES,
    'brain_negative_samples_per_epoch': BRAIN_NEGATIVE_SAMPLES,
    'positive_sampling': (
        'every_nonempty_train_subject_once_then_alternating_tiny_small_extras'
    ),
    'curriculum_phases': [
        {
            'start_epoch': 1,
            'end_epoch': 8,
            'positive_samples': 144,
            'topology_negative_samples': 30,
            'brain_negative_samples': 30,
        },
        {
            'start_epoch': 9,
            'end_epoch': 20,
            'positive_samples': 122,
            'topology_negative_samples': 41,
            'brain_negative_samples': 41,
        },
        {
            'start_epoch': 21,
            'end_epoch': 60,
            'positive_samples': 102,
            'topology_negative_samples': 51,
            'brain_negative_samples': 51,
        },
    ],
    'topology_negative_relations_priority': [
        'contralateral_homolog',
        'template_contact_26n',
        'same_unilateral_parent',
    ],
    'negative_requirement': 'exactly_zero_target_voxels',
    'clinical_window_recalibration': {
        'cta': [0.0, 90.0],
        'cbf': [0.0, 35.0],
        'cbv': [0.0, 10.0],
        'mtt': [0.0, 20.0],
        'tmax': [0.0, 7.0],
    },
    'case_loss_weight': (
        'sqrt(train_median_nonempty_volume/case_volume)_clipped'
    ),
    'case_loss_weight_bounds': [CASE_WEIGHT_MIN, CASE_WEIGHT_MAX],
    'batch_size': 2,
    'gradient_accumulation_steps': 2,
    'effective_batch_size': 4,
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'early_stopping_patience_validations': 18,
    'early_stopping_minimum_epoch': 30,
    'optimizer': 'AdamW',
    'learning_rate': 8e-5,
    'weight_decay': 2e-3,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'ema_decay': 0.99,
    'validation_model': 'exponential_moving_average',
    'intensity_gain_range': [0.90, 1.10],
    'intensity_offset_range': [-0.03, 0.03],
    'noise_probability': 0.65,
    'maximum_noise_standard_deviation': 0.02,
    'gamma_probability': 0.50,
    'gamma_range': [0.85, 1.15],
    'spatial_affine_probability': 0.70,
    'spatial_rotation_degrees': 10.0,
    'spatial_scale_range': [0.90, 1.10],
    'spatial_translation_fraction_xy': 0.06,
    'spatial_translation_fraction_z': 0.03,
    'left_right_reflection': False,
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': 0.20,
    'main_loss_weights': {
        'case_weighted_soft_dice': 0.35,
        'case_weighted_hard_background_bce': 0.30,
        'case_weighted_focal_tversky': 0.35,
    },
    'focal_tversky_alpha_false_positive': 0.35,
    'focal_tversky_beta_false_negative': 0.65,
    'focal_tversky_gamma': 0.75,
    'focal_tversky_error_floor': 1e-6,
    'ohem_background_to_foreground_ratio': 3.0,
    'ohem_minimum_background_voxels': 4096,
    'ohem_maximum_background_voxels': 65536,
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'territory_node_loss_weight': 0.04,
    'territory_positive_weight': 4.0,
    'threshold_candidates': [
        round(float(value), 3)
        for value in np.arange(0.05, 0.9501, 0.025)
    ],
    'model_selection': 'validation_mean_subject_dice_only',
    'test_access_during_training': False,
}

all_checks = {
    **upstream_checks,
    **manifest_checks,
    **sampling_checks,
    **window_checks,
    'patch_dimensions_divisible_by_16': all(
        size % 16 == 0 for size in PATCH_SIZE
    ),
    'epoch_has_204_samples': EPOCH_SAMPLES == 204,
    'architecture_parameter_count_frozen': (
        EXPECTED_PARAMETERS == 15_052_628
    ),
    'validation_threshold_grid_contains_v2_optimum': (
        0.65 in training_policy['threshold_candidates']
    ),
    'test_targets_not_read': True,
    'study_trainable_parameters_are_zero': True,
}
preflight_passed = bool(all(all_checks.values()))

atomic_write_json(training_policy, POLICY_PATH)
atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': V4_PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'v4_training_authorized': preflight_passed,
        'checks': all_checks,
        'training_policy': training_policy,
        'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
        'verified_manifest_sha256': manifest_sha256,
        'graph_schema_path': str(GRAPH_SCHEMA_PATH),
        'graph_schema_sha256': schema_sha256,
        'scaler_path': str(SCALER_PATH),
        'scaler_sha256': scaler_sha256,
        'median_nonempty_train_lesion_volume_ml': median_nonempty_volume_ml,
        'minimum_train_target_inside_crop_fraction': float(
            target_audit_df['target_inside_crop_fraction'].min()
        ),
        'target_arrays_read_splits': ['train'],
        'test_targets_read': False,
        'study_trainable_parameters': 0,
        'expected_model_parameters': EXPECTED_PARAMETERS,
        'sampling_audit_path': str(SAMPLING_AUDIT_PATH),
        'window_audit_path': str(WINDOW_AUDIT_PATH),
    },
    GATE_PATH,
)

print('\nTrain lesion-volume and capped-weight audit:')
display(
    target_audit_df.groupby('lesion_volume_bin', dropna=False)
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('lesion_volume_ml', 'median'),
        minimum_target_inside_crop_fraction=(
            'target_inside_crop_fraction', 'min'
        ),
        mean_case_loss_weight=('positive_case_loss_weight', 'mean'),
        maximum_case_loss_weight=('positive_case_loss_weight', 'max'),
    )
    .reset_index()
)

print('\nV4 stage-1 sampling audit:')
display(
    sampling_audit_df.groupby(
        ['sample_type', 'lesion_volume_bin'], dropna=False
    )
    .agg(
        patches=('sample_index', 'count'),
        minimum_target_voxels=('patch_target_voxels', 'min'),
        maximum_target_voxels=('patch_target_voxels', 'max'),
        mean_case_loss_weight=('case_loss_weight', 'mean'),
    )
    .reset_index()
)

print('\nTopology-negative relation counts:')
display(
    sampling_audit_df[
        sampling_audit_df['sample_type'] == 'topology_negative'
    ]['topology_relation'].value_counts().rename_axis(
        'topology_relation'
    ).reset_index(name='patches')
)

print('\nClinical-window recalibration audit:')
display(
    window_audit_df.groupby('channel_name')
    .agg(
        subjects=('subject_id', 'count'),
        global_minimum=('minimum', 'min'),
        global_maximum=('maximum', 'max'),
        mean_changed_fraction=('changed_fraction', 'mean'),
        all_finite=('all_finite', 'all'),
    )
    .reset_index()
)

print('\nV4 preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in all_checks.items()
]))

if not preflight_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V4 SAMPLER/LOSS PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 V4 SCALE-BALANCED OHEM PREFLIGHT PASSED')
print('=' * 80)
print(f'Train targets audited:              {len(target_audit_df)}')
print('Validation targets read:            False')
print('Test targets read:                  False')
print(f'Patch size:                         {PATCH_SIZE}')
print(f'Samples per epoch:                  {EPOCH_SAMPLES}')
print('Stage-1 positive/topology/brain:    144 / 30 / 30')
print(f'Stage-1 positive patch fraction:    {POSITIVE_SAMPLES / EPOCH_SAMPLES:.3f}')
print('Negative target voxels:             exactly zero')
print(f'Case-loss weight bounds:            {CASE_WEIGHT_MIN:.2f}..{CASE_WEIGHT_MAX:.2f}')
print('Clinical windows:                   CTA/CBF/MTT/Tmax recalibrated')
print('Novel curriculum:                   subject-covered scale curriculum + OHEM')
print('Scientific core changed:            False')
print('Study trainable parameters:         0 (preflight only)')
print(f'Expected model parameters:          {EXPECTED_PARAMETERS:,}')
print(f'Policy:                             {POLICY_PATH}')
print(f'Preflight gate:                     {GATE_PATH}')
print('V4 training authorized:             True')
print('\nStop here and check every table before running the v4 training block.')


In [ ]:
import sys
import os
import gc
import copy
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v4_scale_balanced_ohem_model_recheck'
CODE_REVISION = 'r1_subject_covered_staged_ohem'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (112, 112, 48)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(
            'Trial8 v3 requires a CUDA GPU with native bfloat16 support.'
        )

MIXED_PRECISION_DTYPE = torch.bfloat16
MIXED_PRECISION_NAME = 'bfloat16'
AMP_SCALER_ENABLED = False

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=MIXED_PRECISION_DTYPE,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=MIXED_PRECISION_DTYPE,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'model_training_authorized': preflight_passed,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Model training authorized:          True')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: build the isolated Trial8 training cell from this frozen preflight.')






print('\n' + '=' * 80)
print('TRIAL8 TARGET-AWARE TRAINING PREFLIGHT')
print('=' * 80)
print('Target access policy: train and validation only; test remains sealed.')

target_rows = []
target_arrays_read_splits = set()
for row in manifest_df[
    manifest_df['dev_split'].isin(['train', 'val'])
].sort_values(['dev_split', 'subject_id']).to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    source_path = Path(str(row['vascufate_v2_npz_path']))
    if not source_path.exists():
        raise FileNotFoundError(f'{subject_id}: frozen NPZ is missing.')
    with np.load(source_path, allow_pickle=False) as data:
        required = {
            'target_binary',
            'target_soft',
            'binary_loss_weight',
            'crop_mask',
            'target_spacing',
            'subject_id',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        spatial_shape = tuple(int(value) for value in target_binary.shape)
        crop_mask = np.asarray(data['crop_mask']) > 0
        binary_loss_weight_value = float(
            np.asarray(data['binary_loss_weight']).item()
        )
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        stored_subject = scalar_string(data['subject_id'])
    target_arrays_read_splits.add(split)
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: stored subject identifier mismatch.')
    if target_soft.shape != target_binary.shape or crop_mask.shape != target_binary.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not set(int(value) for value in np.unique(target_binary)).issubset({0, 1}):
        raise RuntimeError(f'{subject_id}: binary target is not binary.')
    if (
        not np.isfinite(target_soft).all()
        or target_soft.min() < -1e-4
        or target_soft.max() > 1.0001
    ):
        raise RuntimeError(f'{subject_id}: soft target is invalid.')
    if binary_loss_weight_value not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    target_voxels = int(target_binary.sum())
    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
        'spatial_shape': str(spatial_shape),
        'binary_voxels': target_voxels,
        'binary_volume_ml': float(target_voxels * voxel_volume_ml),
        'soft_sum': float(target_soft.sum(dtype=np.float64)),
        'binary_loss_weight': binary_loss_weight_value,
        'target_inside_crop_fraction': 1.0 if target_voxels == 0 else float(
            (target_binary & crop_mask).sum() / target_voxels
        ),
        'all_finite': True,
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_audit_checks = {
    'exactly_102_train_targets_audited': int(
        (target_audit_df['dev_split'] == 'train').sum()
    )
    == 102,
    'exactly_23_validation_targets_audited': int(
        (target_audit_df['dev_split'] == 'val').sum()
    )
    == 23,
    'test_targets_not_opened': target_arrays_read_splits == {'train', 'val'},
    'all_target_arrays_finite': bool(target_audit_df['all_finite'].all()),
    'target_crop_support_fractions_valid': bool(
        target_audit_df['target_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'train_contains_nonempty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] > 0)
        ).any()
    ),
    'train_contains_empty_targets': bool(
        (
            (target_audit_df['dev_split'] == 'train')
            & (target_audit_df['binary_voxels'] == 0)
        ).any()
    ),
}
if not all(target_audit_checks.values()):
    failed = [
        name for name, passed in target_audit_checks.items() if not passed
    ]
    raise RuntimeError('Target audit failed: ' + ', '.join(failed))

print('\nTarget audit by split:')
display(
    target_audit_df.groupby('dev_split').agg(
        subjects=('subject_id', 'count'),
        empty_targets=('binary_voxels', lambda values: int((values == 0).sum())),
        minimum_volume_ml=('binary_volume_ml', 'min'),
        median_volume_ml=('binary_volume_ml', 'median'),
        maximum_volume_ml=('binary_volume_ml', 'max'),
    ).reset_index()
)
print('\nTarget audit by training lesion bin:')
display(
    target_audit_df[target_audit_df['dev_split'] == 'train']
    .groupby('lesion_volume_bin')
    .agg(
        subjects=('subject_id', 'count'),
        median_volume_ml=('binary_volume_ml', 'median'),
    )
    .reset_index()
)






def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)

    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(f'{row["subject_id"]}: atlas labels outside 0..30.')
    return labels, True


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size, spatial_axes):
    source_slices = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source_slices[axis] = spatial_slice
    cropped = array[tuple(source_slices)]
    return center_crop_or_pad(cropped, patch_size, spatial_axes)


def deterministic_patch_center(target, crop_mask, seed, positive_probability=0.75):
    rng = np.random.default_rng(seed)
    positive_coordinates = np.argwhere(target > 0)
    use_positive = (
        len(positive_coordinates) > 0
        and float(rng.random()) < positive_probability
    )
    if use_positive:
        center = positive_coordinates[
            int(rng.integers(0, len(positive_coordinates)))
        ].astype(np.int64)
        jitter = np.asarray([
            int(rng.integers(-16, 17)),
            int(rng.integers(-16, 17)),
            int(rng.integers(-6, 7)),
        ])
        center = center + jitter
        source = 'lesion_jitter'
    else:
        candidates = np.argwhere(crop_mask > 0)
        if len(candidates) == 0:
            center = np.asarray(target.shape) // 2
        else:
            center = candidates[int(rng.integers(0, len(candidates)))]
        source = 'brain_random'
    center = np.clip(center, 0, np.asarray(target.shape) - 1)
    return center, source


manifest_by_subject = {
    str(row['subject_id']): row
    for row in manifest_df.to_dict('records')
}
train_target_df = target_audit_df[target_audit_df['dev_split'] == 'train']
sampling_subjects = []
for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
    matches = train_target_df[
        train_target_df['lesion_volume_bin'] == lesion_bin
    ].sort_values('subject_id')
    if len(matches):
        sampling_subjects.append(str(matches.iloc[0]['subject_id']))
sampling_subjects = list(dict.fromkeys(sampling_subjects))

patch_rows = []
for subject_offset, subject_id in enumerate(sampling_subjects):
    row = manifest_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    for repeat in range(4):
        center, source = deterministic_patch_center(
            target,
            crop_mask,
            SEED + subject_offset * 100 + repeat,
        )
        slices = patch_slices_from_center(target.shape, center, PATCH_SIZE)
        patch_target = extract_patch(
            target, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        patch_rows.append({
            'subject_id': subject_id,
            'lesion_volume_bin': str(row.get('lesion_volume_bin', 'unknown')),
            'repeat': repeat,
            'center_source': source,
            'full_target_voxels': int(target.sum()),
            'patch_target_voxels': int(patch_target.sum()),
            'positive_patch_contains_lesion': (
                source != 'lesion_jitter' or bool(patch_target.any())
            ),
        })

patch_audit_df = pd.DataFrame(patch_rows)
atomic_write_csv(patch_audit_df, PATCH_AUDIT_PATH)
patch_checks = {
    'all_requested_lesion_bins_sampled': len(sampling_subjects) >= 5,
    'four_patches_per_selected_subject': len(patch_audit_df)
    == 4 * len(sampling_subjects),
    'all_lesion_centered_patches_contain_lesion': bool(
        patch_audit_df['positive_patch_contains_lesion'].all()
    ),
    'empty_subject_sampling_supported': bool(
        (patch_audit_df['full_target_voxels'] == 0).any()
    ),
}
if not all(patch_checks.values()):
    failed = [name for name, passed in patch_checks.items() if not passed]
    raise RuntimeError('Patch-sampling checks failed: ' + ', '.join(failed))

print('\nLesion-aware patch audit:')
display(
    patch_audit_df.groupby(
        ['lesion_volume_bin', 'center_source'], dropna=False
    ).agg(
        patches=('subject_id', 'count'),
        minimum_patch_target_voxels=('patch_target_voxels', 'min'),
        maximum_patch_target_voxels=('patch_target_voxels', 'max'),
    ).reset_index()
)








V4_DICE_WEIGHT = 0.35
V4_OHEM_BCE_WEIGHT = 0.30
V4_TVERSKY_WEIGHT = 0.35
V4_TVERSKY_ALPHA = 0.35
V4_TVERSKY_BETA = 0.65
V4_TVERSKY_GAMMA = 0.75
V4_TVERSKY_ERROR_FLOOR = 1e-6
V4_OHEM_BACKGROUND_TO_FOREGROUND_RATIO = 3.0
V4_OHEM_MINIMUM_BACKGROUND_VOXELS = 4096
V4_OHEM_MAXIMUM_BACKGROUND_VOXELS = 65536

def normalized_case_weights(case_weight, batch_size, device):
    if case_weight is None:
        return torch.ones(batch_size, device=device, dtype=torch.float32)
    return case_weight.reshape(-1).to(device=device, dtype=torch.float32)


def weighted_case_mean(values, case_weight):
    values = values.reshape(-1)
    weights = case_weight.reshape(-1).to(values.dtype)
    return (values * weights).sum() / weights.sum().clamp_min(1e-6)


def soft_dice_loss(logits, target, case_weight=None, epsilon=1.0):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    per_case = 1.0 - (2.0 * intersection + epsilon) / (
        denominator + epsilon
    )
    weights = normalized_case_weights(
        case_weight, logits.shape[0], logits.device
    )
    return weighted_case_mean(per_case, weights)


def hard_background_mined_bce(
    logits,
    soft_target,
    binary_target,
    case_weight=None,
    background_to_foreground_ratio=V4_OHEM_BACKGROUND_TO_FOREGROUND_RATIO,
    minimum_background_voxels=V4_OHEM_MINIMUM_BACKGROUND_VOXELS,
    maximum_background_voxels=V4_OHEM_MAXIMUM_BACKGROUND_VOXELS,
):
    
    dilated = F.max_pool3d(binary_target, kernel_size=3, stride=1, padding=1)
    eroded = -F.max_pool3d(
        -binary_target, kernel_size=3, stride=1, padding=1
    )
    boundary = (dilated - eroded).clamp(0.0, 1.0)
    interpolation_uncertainty = 4.0 * soft_target * (1.0 - soft_target)
    weights = 1.0 + 2.0 * boundary + interpolation_uncertainty
    raw_loss = F.binary_cross_entropy_with_logits(
        logits, soft_target, reduction='none'
    ) * weights

    per_case = []
    for case_index in range(logits.shape[0]):
        foreground = (
            (soft_target[case_index] > 0.01)
            | (binary_target[case_index] > 0)
        )
        foreground_loss = raw_loss[case_index][foreground]
        background_loss = raw_loss[case_index][~foreground]
        desired_background = max(
            int(minimum_background_voxels),
            int(math.ceil(
                max(int(foreground_loss.numel()), 1)
                * float(background_to_foreground_ratio)
            )),
        )
        selected_background = min(
            int(background_loss.numel()),
            int(maximum_background_voxels),
            desired_background,
        )
        if selected_background > 0:
            hard_background_loss = torch.topk(
                background_loss,
                k=selected_background,
                largest=True,
                sorted=False,
            ).values
        else:
            hard_background_loss = background_loss
        selected_voxels = int(foreground_loss.numel()) + selected_background
        per_case.append(
            (foreground_loss.sum() + hard_background_loss.sum())
            / max(selected_voxels, 1)
        )
    per_case = torch.stack(per_case)
    case_weights = normalized_case_weights(
        case_weight, logits.shape[0], logits.device
    )
    return weighted_case_mean(per_case, case_weights)


def focal_tversky_loss(
    logits,
    target,
    alpha=V4_TVERSKY_ALPHA,
    beta=V4_TVERSKY_BETA,
    gamma=V4_TVERSKY_GAMMA,
    error_floor=V4_TVERSKY_ERROR_FLOOR,
):
    probability = torch.sigmoid(logits)
    dimensions = tuple(range(2, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error = (1.0 - tversky).clamp(0.0, 1.0)
    
    
    
    
    stabilized = error.clamp_min(error_floor)
    return stabilized.pow(gamma) - float(error_floor ** gamma)


def segmentation_loss(
    logits,
    soft_target,
    binary_target,
    hard_loss_weight,
    case_loss_weight=None,
):
    
    
    logits = logits.float()
    soft_target = soft_target.float()
    binary_target = binary_target.float()
    case_weights = normalized_case_weights(
        case_loss_weight, logits.shape[0], logits.device
    )
    dice = soft_dice_loss(
        logits, soft_target, case_weight=case_weights
    )
    hard_background_bce = hard_background_mined_bce(
        logits,
        soft_target,
        binary_target,
        case_weight=case_weights,
    )
    hard_tversky_per_case = focal_tversky_loss(logits, binary_target)
    
    
    hard_loss_weight = (
        hard_loss_weight.reshape(-1, 1)
        * case_weights.reshape(-1, 1)
    )
    hard_tversky = (
        hard_tversky_per_case * hard_loss_weight
    ).sum() / hard_loss_weight.sum().clamp_min(1.0)
    total = (
        V4_DICE_WEIGHT * dice
        + V4_OHEM_BCE_WEIGHT * hard_background_bce
        + V4_TVERSKY_WEIGHT * hard_tversky
    )
    return total, {
        'soft_dice': dice,
        'hard_background_bce': hard_background_bce,
        'hard_focal_tversky': hard_tversky,
    }


def territory_presence_target(binary_target, region_map):
    targets = []
    for label in range(1, NUM_TERRITORY_NODES + 1):
        targets.append(
            ((binary_target > 0) & (region_map == label)).any().astype(np.float32)
        )
    return np.asarray(targets, dtype=np.float32)


eligible_real_df = train_target_df[
    (train_target_df['binary_voxels'] > 0)
    & (train_target_df['binary_loss_weight'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_real_df = eligible_real_df[
    eligible_real_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
candidate_midpoint = len(eligible_real_df) // 2
candidate_order = sorted(
    range(len(eligible_real_df)),
    key=lambda index: (abs(index - candidate_midpoint), index),
)
real_subject = None
for candidate_index in candidate_order:
    candidate_subject = str(
        eligible_real_df.iloc[candidate_index]['subject_id']
    )
    candidate_row = manifest_by_subject[candidate_subject]
    with np.load(
        Path(str(candidate_row['vascufate_v2_npz_path'])),
        allow_pickle=False,
    ) as candidate_data:
        candidate_target = np.asarray(
            candidate_data['target_binary'], dtype=np.uint8
        )
    candidate_region, _ = load_region_map(
        candidate_row, candidate_target.shape
    )
    if territory_presence_target(
        candidate_target, candidate_region
    ).sum() > 0:
        real_subject = candidate_subject
        break
if real_subject is None:
    raise RuntimeError(
        'No atlas-enabled training lesion overlaps a valid territory.'
    )
real_row = manifest_by_subject[real_subject]
real_graph = load_graph_inputs(real_row)
if file_sha256(Path(str(real_row['vascufate_v2_npz_path']))) != str(
    real_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{real_subject}: real-loss NPZ hash changed.')
with np.load(
    Path(str(real_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    real_image = np.asarray(data['image'], dtype=np.float32)
    real_binary = np.asarray(data['target_binary'], dtype=np.float32)
    real_soft = np.asarray(data['target_soft'], dtype=np.float32)
    real_crop_mask = np.asarray(data['crop_mask']) > 0
    real_binary_weight = float(np.asarray(data['binary_loss_weight']).item())
real_region, real_atlas_enabled = load_region_map(
    real_row, real_binary.shape
)
real_territory_target = territory_presence_target(real_binary, real_region)
real_center, real_center_source = deterministic_patch_center(
    real_binary, real_crop_mask, SEED + 9001, positive_probability=1.0
)
real_slices = patch_slices_from_center(
    real_binary.shape, real_center, PATCH_SIZE
)
real_image_patch = extract_patch(
    real_image, real_slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
)
real_binary_patch = extract_patch(
    real_binary, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_soft_patch = extract_patch(
    real_soft, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
real_region_patch = extract_patch(
    real_region, real_slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
)

real_batch = {
    'image': torch.from_numpy(real_image_patch[None]).to(device),
    'region_map': torch.from_numpy(real_region_patch[None]).to(device),
    'territory_features': torch.from_numpy(
        real_graph['territory_features'][None]
    ).to(device),
    'territory_feature_valid': torch.from_numpy(
        real_graph['territory_feature_valid'][None]
    ).to(device),
    'territory_valid': torch.from_numpy(
        real_graph['territory_valid'][None]
    ).to(device),
    'territory_reliability': torch.from_numpy(
        real_graph['territory_reliability'][None]
    ).to(device),
    'vessel_features': torch.from_numpy(
        real_graph['vessel_features'][None]
    ).to(device),
    'vessel_feature_valid': torch.from_numpy(
        real_graph['vessel_feature_valid'][None]
    ).to(device),
    'vessel_valid': torch.from_numpy(real_graph['vessel_valid'][None]).to(device),
    'case_features': torch.from_numpy(real_graph['case_features'][None]).to(device),
    'atlas_enabled': torch.as_tensor(
        [real_atlas_enabled], dtype=torch.bool, device=device
    ),
}
real_soft_tensor = torch.from_numpy(real_soft_patch[None, None]).to(device)
real_binary_tensor = torch.from_numpy(real_binary_patch[None, None]).to(device)
real_hard_weight = torch.as_tensor(
    [real_binary_weight], dtype=torch.float32, device=device
)
real_territory_target_tensor = torch.from_numpy(
    real_territory_target[None]
).to(device)


set_determinism(SEED + 1)
model = VascuFateNet(
    schema=graph_schema,
    image_channels=real_image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
amp_scaler = torch.amp.GradScaler(
    device.type, enabled=AMP_SCALER_ENABLED
)
before_step = model.segmentation_head.weight.detach().clone()
model.train()
optimizer.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=MIXED_PRECISION_DTYPE,
    enabled=device.type == 'cuda',
):
    real_output = model(**real_batch, force_atlas_dropout=False)
    main_loss, main_parts = segmentation_loss(
        real_output['logits'],
        real_soft_tensor,
        real_binary_tensor,
        real_hard_weight,
    )
    half_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        real_output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        real_hard_weight,
    )
    quarter_soft = F.interpolate(
        real_soft_tensor,
        size=real_output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        real_binary_tensor,
        real_output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        real_output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        real_hard_weight,
    )
    node_mask = (
        real_batch['territory_valid'].bool()
        & real_output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        real_output['territory_node_logits'].float(),
        real_territory_target_tensor.float(),
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0, device=device
        ),
    )
    if bool(node_mask.any()):
        node_loss = node_bce[node_mask].mean()
    else:
        node_loss = real_output['territory_node_logits'].sum() * 0.0
    total_real_loss = (
        main_loss
        + 0.15 * half_loss
        + 0.10 * quarter_loss
        + 0.04 * node_loss
    )

amp_scaler.scale(total_real_loss).backward()
amp_scaler.unscale_(optimizer)
real_gradient_norm = float(
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0).item()
)
amp_scaler.step(optimizer)
amp_scaler.update()
optimizer.zero_grad(set_to_none=True)
parameter_update = float(
    (model.segmentation_head.weight.detach() - before_step).abs().max().item()
)



empty_edge_logits = torch.full(
    (1, 1, 4, 4, 4),
    -100.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
empty_edge_target = torch.zeros_like(empty_edge_logits)
empty_edge_loss = focal_tversky_loss(
    empty_edge_logits, empty_edge_target
).mean()
empty_edge_gradient = torch.autograd.grad(
    empty_edge_loss, empty_edge_logits
)[0]
empty_edge_gradient_finite = bool(
    torch.isfinite(empty_edge_gradient).all().item()
)

loss_values = {
    'subject_id': real_subject,
    'patch_center_source': real_center_source,
    'patch_target_voxels': int(real_binary_patch.sum()),
    'main_loss': float(main_loss.detach().cpu()),
    'soft_dice_loss': float(main_parts['soft_dice'].detach().cpu()),
    'hard_background_bce_loss': float(
        main_parts['hard_background_bce'].detach().cpu()
    ),
    'hard_focal_tversky_loss': float(
        main_parts['hard_focal_tversky'].detach().cpu()
    ),
    'half_resolution_loss': float(half_loss.detach().cpu()),
    'quarter_resolution_loss': float(quarter_loss.detach().cpu()),
    'territory_node_loss': float(node_loss.detach().cpu()),
    'total_loss': float(total_real_loss.detach().cpu()),
    'gradient_norm_before_clipping': real_gradient_norm,
    'segmentation_head_maximum_update': parameter_update,
    'empty_target_edge_loss': float(empty_edge_loss.detach().cpu()),
    'empty_target_edge_maximum_gradient': float(
        empty_edge_gradient.detach().abs().max().cpu()
    ),
}
atomic_write_json(loss_values, LOSS_AUDIT_PATH)
loss_checks = {
    'real_patch_contains_lesion': int(real_binary_patch.sum()) > 0,
    'all_real_loss_components_finite': all(
        math.isfinite(float(value))
        for key, value in loss_values.items()
        if key.endswith('_loss') or key == 'total_loss'
    ),
    'real_gradient_norm_positive_and_finite': math.isfinite(real_gradient_norm)
    and real_gradient_norm > 0,
    'optimizer_step_changed_parameters': parameter_update > 0,
    'empty_target_focal_tversky_gradient_finite': (
        empty_edge_gradient_finite
    ),
    'empty_target_focal_tversky_loss_near_zero': (
        math.isfinite(float(empty_edge_loss.detach().cpu()))
        and abs(float(empty_edge_loss.detach().cpu())) < 1e-6
    ),
    'territory_target_has_30_nodes': real_territory_target.shape == (30,),
    'territory_target_contains_positive_node': real_territory_target.sum() > 0,
    'parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
}
if not all(loss_checks.values()):
    failed = [name for name, passed in loss_checks.items() if not passed]
    raise RuntimeError('Real-loss optimizer preflight failed: ' + ', '.join(failed))

print('\nReal-loss disposable optimizer-step audit:')
display(pd.DataFrame([
    {'metric': name, 'value': value} for name, value in loss_values.items()
]))
print('\nParameter count after model reinitialization:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': parameter_count(model)},
    {'component': 'trainable', 'parameters': parameter_count(model, True)},
]))






def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


@torch.no_grad()
def sliding_window_probability(model, row, image, region_map, graph_inputs):
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, PATCH_SIZE)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(PATCH_SIZE)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, PATCH_SIZE)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    model.eval()
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index : location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(
                np.repeat(graph_inputs['territory_features'][None], count, axis=0)
            ).to(device),
            'territory_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'territory_valid': torch.from_numpy(
                np.repeat(graph_inputs['territory_valid'][None], count, axis=0)
            ).to(device),
            'territory_reliability': torch.from_numpy(
                np.repeat(
                    graph_inputs['territory_reliability'][None], count, axis=0
                )
            ).to(device),
            'vessel_features': torch.from_numpy(
                np.repeat(graph_inputs['vessel_features'][None], count, axis=0)
            ).to(device),
            'vessel_feature_valid': torch.from_numpy(
                np.repeat(
                    graph_inputs['vessel_feature_valid'][None], count, axis=0
                )
            ).to(device),
            'vessel_valid': torch.from_numpy(
                np.repeat(graph_inputs['vessel_valid'][None], count, axis=0)
            ).to(device),
            'case_features': torch.from_numpy(
                np.repeat(graph_inputs['case_features'][None], count, axis=0)
            ).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.autocast(
            device_type=device.type,
            dtype=MIXED_PRECISION_DTYPE,
            enabled=device.type == 'cuda',
        ):
            logits = model(
                **inference_batch, force_atlas_dropout=False
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + PATCH_SIZE[0]),
                slice(y, y + PATCH_SIZE[1]),
                slice(z, z + PATCH_SIZE[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    return probability[original_slices]


eligible_validation_df = target_audit_df[
    (target_audit_df['dev_split'] == 'val')
    & (target_audit_df['binary_voxels'] > 0)
].sort_values(['binary_volume_ml', 'subject_id'])
eligible_validation_df = eligible_validation_df[
    eligible_validation_df['subject_id'].map(
        lambda subject_id: as_bool(
            manifest_by_subject[str(subject_id)]['trial8_atlas_enabled']
        )
    )
]
validation_subject = str(
    eligible_validation_df.iloc[len(eligible_validation_df) // 2]['subject_id']
)
validation_row = manifest_by_subject[validation_subject]
validation_graph = load_graph_inputs(validation_row)
if file_sha256(Path(str(validation_row['vascufate_v2_npz_path']))) != str(
    validation_row['trial8_npz_sha256']
):
    raise RuntimeError(f'{validation_subject}: validation NPZ hash changed.')
with np.load(
    Path(str(validation_row['vascufate_v2_npz_path'])), allow_pickle=False
) as data:
    validation_image = np.asarray(data['image'], dtype=np.float32)
    validation_target = np.asarray(data['target_binary'], dtype=np.uint8)
validation_region, _ = load_region_map(
    validation_row, validation_target.shape
)
validation_probability = sliding_window_probability(
    model,
    validation_row,
    validation_image,
    validation_region,
    validation_graph,
)
threshold_rows = []
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    prediction = validation_probability >= threshold
    intersection = int((prediction & (validation_target > 0)).sum())
    predicted = int(prediction.sum())
    truth = int(validation_target.sum())
    dice = (
        1.0
        if predicted + truth == 0
        else (2.0 * intersection) / (predicted + truth)
    )
    recall = 1.0 if truth == 0 else intersection / truth
    precision = 1.0 if predicted == 0 and truth == 0 else (
        intersection / max(predicted, 1)
    )
    threshold_rows.append({
        'subject_id': validation_subject,
        'threshold': threshold,
        'dice': dice,
        'precision': precision,
        'recall': recall,
        'predicted_voxels': predicted,
        'target_voxels': truth,
    })
validation_metric_df = pd.DataFrame(threshold_rows)
atomic_write_csv(validation_metric_df, VALIDATION_AUDIT_PATH)
validation_checks = {
    'validation_probability_shape_matches': validation_probability.shape
    == validation_target.shape,
    'validation_probability_finite': bool(
        np.isfinite(validation_probability).all()
    ),
    'validation_probability_in_unit_interval': bool(
        validation_probability.min() >= 0
        and validation_probability.max() <= 1
    ),
    'five_thresholds_evaluated': len(validation_metric_df) == 5,
    'validation_metrics_finite': bool(
        np.isfinite(
            validation_metric_df[['dice', 'precision', 'recall']].to_numpy()
        ).all()
    ),
    'test_targets_still_not_opened': target_arrays_read_splits == {
        'train',
        'val',
    },
}
if not all(validation_checks.values()):
    failed = [name for name, passed in validation_checks.items() if not passed]
    raise RuntimeError(
        'Sliding-window validation preflight failed: ' + ', '.join(failed)
    )

print('\nDisposable full-volume validation smoke metrics:')
display(validation_metric_df)






V4_PREFLIGHT_VERSION = 'v4_scale_balanced_ohem_curriculum'
V4_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / V4_PREFLIGHT_VERSION
)
V4_GATE_PATH = V4_PREFLIGHT_ROOT / 'v4_sampler_loss_preflight_gate.json'
V4_TARGET_AUDIT_PATH = V4_PREFLIGHT_ROOT / 'train_target_audit.csv'
if not V4_GATE_PATH.exists() or not V4_TARGET_AUDIT_PATH.exists():
    raise FileNotFoundError(
        'The standalone Trial8 v4 sampler/loss preflight must be run first.'
    )

v4_gate_sha256 = file_sha256(V4_GATE_PATH)
v4_gate = json.loads(V4_GATE_PATH.read_text(encoding='utf-8'))
if v4_gate.get('preflight_passed') is not True:
    raise RuntimeError('Trial8 v4 sampler/loss preflight did not pass.')
if v4_gate.get('v4_training_authorized') is not True:
    raise RuntimeError('Trial8 v4 training is not authorized.')
if v4_gate.get('test_targets_read') is not False:
    raise RuntimeError('Trial8 v4 preflight reports test-target access.')
if v4_gate.get('expected_model_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Trial8 v4 expected parameter count changed.')
if v4_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Trial8 v4 manifest hash changed.')
if v4_gate.get('graph_schema_sha256') != safe_schema_sha256:
    raise RuntimeError('Trial8 v4 graph-schema hash changed.')
if v4_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Trial8 v4 scaler hash changed.')

policy = v4_gate['training_policy']
policy_checks = {
    'patch_size_matches': tuple(policy['patch_size']) == PATCH_SIZE,
    'samples_per_epoch_is_204': int(policy['samples_per_epoch']) == 204,
    'stage1_positive_samples_are_144': int(
        policy['positive_samples_per_epoch']
    ) == 144,
    'stage1_topology_negative_samples_are_30': int(
        policy['topology_negative_samples_per_epoch']
    ) == 30,
    'stage1_brain_negative_samples_are_30': int(
        policy['brain_negative_samples_per_epoch']
    ) == 30,
    'three_curriculum_phases_present': len(
        policy['curriculum_phases']
    ) == 3,
    'curriculum_phase_counts_match': policy['curriculum_phases'] == [
        {
            'start_epoch': 1,
            'end_epoch': 8,
            'positive_samples': 144,
            'topology_negative_samples': 30,
            'brain_negative_samples': 30,
        },
        {
            'start_epoch': 9,
            'end_epoch': 20,
            'positive_samples': 122,
            'topology_negative_samples': 41,
            'brain_negative_samples': 41,
        },
        {
            'start_epoch': 21,
            'end_epoch': 60,
            'positive_samples': 102,
            'topology_negative_samples': 51,
            'brain_negative_samples': 51,
        },
    ],
    'ohem_policy_present': (
        float(policy['ohem_background_to_foreground_ratio']) == 3.0
        and int(policy['ohem_minimum_background_voxels']) == 4096
        and int(policy['ohem_maximum_background_voxels']) == 65536
    ),
    'main_loss_policy_matches': (
        float(policy['main_loss_weights']['case_weighted_soft_dice'])
        == V4_DICE_WEIGHT
        and float(policy['main_loss_weights'][
            'case_weighted_hard_background_bce'
        ]) == V4_OHEM_BCE_WEIGHT
        and float(policy['main_loss_weights'][
            'case_weighted_focal_tversky'
        ]) == V4_TVERSKY_WEIGHT
        and float(policy['focal_tversky_alpha_false_positive'])
        == V4_TVERSKY_ALPHA
        and float(policy['focal_tversky_beta_false_negative'])
        == V4_TVERSKY_BETA
        and float(policy['focal_tversky_gamma']) == V4_TVERSKY_GAMMA
        and float(policy['focal_tversky_error_floor'])
        == V4_TVERSKY_ERROR_FLOOR
    ),
    'negative_requirement_is_zero_target': (
        policy['negative_requirement'] == 'exactly_zero_target_voxels'
    ),
    'scientific_core_unchanged': policy['scientific_core_changed'] is False,
    'left_right_reflection_disabled': (
        policy['left_right_reflection'] is False
    ),
    'test_access_disabled': policy['test_access_during_training'] is False,
    'early_stopping_waits_for_stage3': int(
        policy['early_stopping_minimum_epoch']
    ) == 30,
}
if not all(policy_checks.values()):
    failed = [name for name, passed in policy_checks.items() if not passed]
    raise RuntimeError('Frozen v4 policy changed: ' + ', '.join(failed))

model_recheck_checks = {
    **frozen_architecture_checks,
    **target_audit_checks,
    **patch_checks,
    **loss_checks,
    **validation_checks,
    **policy_checks,
    'target_free_model_preflight_passed': preflight_passed,
    'model_parameter_count_unchanged': parameter_count(model)
    == EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameter_count_unchanged': parameter_count(model, True)
    == EXPECTED_TOTAL_PARAMETERS,
    'target_access_is_train_and_validation_only': target_arrays_read_splits
    == {'train', 'val'},
}
if not all(model_recheck_checks.values()):
    failed = [
        name for name, passed in model_recheck_checks.items() if not passed
    ]
    raise RuntimeError('Trial8 v4 model recheck failed: ' + ', '.join(failed))

print('\nTrial8 v4 model/policy recheck:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in model_recheck_checks.items()
]))


del model, optimizer, amp_scaler, real_output, real_batch
del real_soft_tensor, real_binary_tensor, real_territory_target_tensor
del normal_output, dropped_output, gradient_output, batch
del smoke_loss, total_real_loss, main_loss, half_loss, quarter_loss
del node_loss, node_bce, before_step
del validation_probability, validation_image, validation_target
del validation_region, validation_graph
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()





TRAINING_VERSION = 'v4_scale_balanced_ohem_curriculum_bf16'
TRAINING_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / TRAINING_VERSION
)
LAST_CHECKPOINT_PATH = TRAINING_ROOT / 'last_checkpoint.pt'
BEST_CHECKPOINT_PATH = TRAINING_ROOT / 'best_checkpoint.pt'
HISTORY_PATH = TRAINING_ROOT / 'training_history.csv'
VALIDATION_ROOT = TRAINING_ROOT / 'validation'
TRAINING_CONFIG_PATH = TRAINING_ROOT / 'training_config.json'
TRAINING_STATE_PATH = TRAINING_ROOT / 'training_state.json'
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = int(policy['batch_size'])
ACCUMULATION_STEPS = int(policy['gradient_accumulation_steps'])
MAX_EPOCHS = int(policy['maximum_epochs'])
PATIENCE_VALIDATIONS = int(policy['early_stopping_patience_validations'])
EARLY_STOPPING_MINIMUM_EPOCH = int(policy['early_stopping_minimum_epoch'])
BASE_LEARNING_RATE = float(policy['learning_rate'])
WEIGHT_DECAY = float(policy['weight_decay'])
GRADIENT_CLIP_NORM = float(policy['gradient_clip_norm'])
EMA_DECAY = float(policy['ema_decay'])
GRAPH_FEATURE_MASK_PROBABILITY = float(
    policy['graph_feature_mask_probability']
)
THRESHOLDS = np.asarray(
    policy['threshold_candidates'], dtype=np.float64
)
CURRICULUM_PHASES = list(policy['curriculum_phases'])
SAMPLES_PER_EPOCH = int(policy['samples_per_epoch'])
CASE_WEIGHT_MIN, CASE_WEIGHT_MAX = [
    float(value) for value in policy['case_loss_weight_bounds']
]
MEDIAN_NONEMPTY_VOLUME_ML = float(
    v4_gate['median_nonempty_train_lesion_volume_ml']
)
NUM_WORKERS = 2
IMPROVEMENT_EPSILON = 1e-4
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
MAX_CENTER_CANDIDATES = 512

territory_edge_index_numpy = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type_numpy = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)
RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}

v4_target_audit_df = pd.read_csv(V4_TARGET_AUDIT_PATH)
if len(v4_target_audit_df) != 102:
    raise RuntimeError('V4 train-target audit no longer has 102 subjects.')
target_meta_by_subject = (
    v4_target_audit_df.set_index('subject_id').to_dict('index')
)

train_rows = (
    manifest_df[manifest_df['dev_split'] == 'train']
    .sort_values('subject_id')
    .to_dict('records')
)
validation_rows = (
    manifest_df[manifest_df['dev_split'] == 'val']
    .sort_values('subject_id')
    .to_dict('records')
)
if len(train_rows) != 102 or len(validation_rows) != 23:
    raise RuntimeError('Frozen train/validation split counts changed.')
train_row_by_subject = {
    str(row['subject_id']): row for row in train_rows
}


def atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


def integral_volume(binary):
    return np.pad(
        np.asarray(binary, dtype=np.int64),
        ((1, 0), (1, 0), (1, 0)),
        mode='constant',
    ).cumsum(0).cumsum(1).cumsum(2)


def box_sum(integral, slices):
    x0, x1 = slices[0].start, slices[0].stop
    y0, y1 = slices[1].start, slices[1].stop
    z0, z1 = slices[2].start, slices[2].stop
    return int(
        integral[x1, y1, z1]
        - integral[x0, y1, z1]
        - integral[x1, y0, z1]
        - integral[x1, y1, z0]
        + integral[x0, y0, z1]
        + integral[x0, y1, z0]
        + integral[x1, y0, z0]
        - integral[x0, y0, z0]
    )


def sampled_coordinates(mask, rng):
    flat = np.flatnonzero(np.asarray(mask, dtype=bool))
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > MAX_CENTER_CANDIDATES:
        flat = rng.choice(
            flat, size=MAX_CENTER_CANDIDATES, replace=False
        )
    return np.column_stack(
        np.unravel_index(flat, mask.shape)
    ).astype(np.int64)


def minimum_overlap_center(target, candidate_mask, rng):
    coordinates = sampled_coordinates(candidate_mask, rng)
    if len(coordinates) == 0:
        return None, None
    integral = integral_volume(target > 0)
    counts = np.asarray([
        box_sum(
            integral,
            patch_slices_from_center(target.shape, center, PATCH_SIZE),
        )
        for center in coordinates
    ], dtype=np.int64)
    minimum = int(counts.min())
    best = np.flatnonzero(counts == minimum)
    selected = int(best[int(rng.integers(0, len(best)))])
    return coordinates[selected], minimum


def load_selection_arrays(row):
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        target = np.asarray(data['target_binary'], dtype=np.uint8)
        crop_mask = np.asarray(data['crop_mask']) > 0
    region_map, atlas_enabled = load_region_map(row, target.shape)
    return target, crop_mask, region_map, atlas_enabled


def topology_candidate_labels(target, region_map):
    lesion_labels = set(
        int(value)
        for value in np.unique(region_map[target > 0])
        if 1 <= int(value) <= NUM_TERRITORY_NODES
    )
    relation_by_label = {}
    relation_priority = {2: 0, 1: 1, 3: 2}
    candidates_by_relation = {2: [], 1: [], 3: []}
    for edge_position in range(territory_edge_index_numpy.shape[1]):
        source = int(territory_edge_index_numpy[0, edge_position]) + 1
        destination = int(
            territory_edge_index_numpy[1, edge_position]
        ) + 1
        relation = int(territory_edge_type_numpy[edge_position])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in candidates_by_relation
        ):
            candidates_by_relation[relation].append(destination)
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    ordered = []
    for relation in [2, 1, 3]:
        ordered.extend(sorted(set(candidates_by_relation[relation])))
    return list(dict.fromkeys(ordered)), relation_by_label


def clinical_window_recalibration(image):
    output = np.asarray(image, dtype=np.float32).copy()
    output[2] = np.clip((output[2] * 1100.0 - 100.0) / 90.0, 0.0, 1.0)
    output[3] = np.clip((output[3] * 150.0) / 35.0, 0.0, 1.0)
    output[5] = np.clip((output[5] * 30.0) / 20.0, 0.0, 1.0)
    output[6] = np.clip((output[6] * 20.0) / 7.0, 0.0, 1.0)
    return output


def augment_training_image(image, rng):
    augmented = np.asarray(image, dtype=np.float32).copy()
    intensity_channels = min(13, augmented.shape[0])
    gains = rng.uniform(
        policy['intensity_gain_range'][0],
        policy['intensity_gain_range'][1],
        size=(intensity_channels, 1, 1, 1),
    )
    offsets = rng.uniform(
        policy['intensity_offset_range'][0],
        policy['intensity_offset_range'][1],
        size=(intensity_channels, 1, 1, 1),
    )
    augmented[:intensity_channels] = (
        augmented[:intensity_channels] * gains + offsets
    )
    if float(rng.random()) < float(policy['noise_probability']):
        noise_scale = float(rng.uniform(
            0.0, policy['maximum_noise_standard_deviation']
        ))
        augmented[:intensity_channels] += rng.normal(
            0.0,
            noise_scale,
            size=augmented[:intensity_channels].shape,
        ).astype(np.float32)
    non_delta_end = min(9, intensity_channels)
    augmented[:non_delta_end] = np.clip(
        augmented[:non_delta_end], 0.0, 1.0
    )
    if (
        non_delta_end > 0
        and float(rng.random()) < float(policy['gamma_probability'])
    ):
        gamma = float(rng.uniform(*policy['gamma_range']))
        augmented[:non_delta_end] = np.power(
            augmented[:non_delta_end], gamma
        ).astype(np.float32)
    if intensity_channels > 9:
        augmented[9:intensity_channels] = np.clip(
            augmented[9:intensity_channels], -1.0, 1.0
        )
    return augmented


def positive_case_weight(subject_id):
    return float(
        target_meta_by_subject[str(subject_id)]['positive_case_loss_weight']
    )


def curriculum_phase_for_epoch(epoch):
    epoch = int(epoch)
    for phase_index, phase in enumerate(CURRICULUM_PHASES, start=1):
        if int(phase['start_epoch']) <= epoch <= int(phase['end_epoch']):
            counts = {
                'positive': int(phase['positive_samples']),
                'topology_negative': int(phase['topology_negative_samples']),
                'brain_negative': int(phase['brain_negative_samples']),
            }
            if sum(counts.values()) != SAMPLES_PER_EPOCH:
                raise RuntimeError(
                    f'V4 curriculum phase {phase_index} does not contain '
                    f'{SAMPLES_PER_EPOCH} samples.'
                )
            return phase_index, counts
    raise RuntimeError(f'No frozen v4 curriculum phase covers epoch {epoch}.')


class Trial8V4PatchDataset(torch.utils.data.Dataset):
    def __init__(self, rows, base_seed):
        self.rows = list(rows)
        self.base_seed = int(base_seed)
        self.epoch = 0
        self.schedule = []
        self.current_phase = 0
        self.current_counts = {}
        self.subjects_by_bin = {
            lesion_bin: sorted([
                subject_id
                for subject_id, metadata in target_meta_by_subject.items()
                if str(metadata['lesion_volume_bin']) == lesion_bin
                and int(metadata['target_voxels']) > 0
            ])
            for lesion_bin in POSITIVE_BINS
        }
        self.topology_pool = sorted([
            str(row['subject_id'])
            for row in self.rows
            if as_bool(row['trial8_atlas_enabled'])
            and int(
                target_meta_by_subject[str(row['subject_id'])][
                    'target_voxels'
                ]
            ) > 0
        ])
        self.brain_pool = sorted(
            [str(row['subject_id']) for row in self.rows],
            key=lambda subject_id: (
                int(target_meta_by_subject[subject_id]['target_voxels']),
                subject_id,
            ),
        )
        self.nonempty_subjects = sorted([
            subject_id
            for subject_id, metadata in target_meta_by_subject.items()
            if int(metadata['target_voxels']) > 0
        ])
        if any(len(values) == 0 for values in self.subjects_by_bin.values()):
            raise RuntimeError('A v4 positive sampling bin is empty.')
        if not self.nonempty_subjects:
            raise RuntimeError('V4 has no nonempty training subjects.')
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        rng = np.random.default_rng(self.base_seed + self.epoch * 100_003)
        self.current_phase, self.current_counts = curriculum_phase_for_epoch(
            self.epoch
        )
        schedule = []
        positive_subjects = list(self.nonempty_subjects)
        if self.current_counts['positive'] < len(positive_subjects):
            raise RuntimeError(
                'V4 curriculum cannot cover every nonempty training subject.'
            )
        extra_index = 0
        extra_bins = ['tiny', 'small']
        while len(positive_subjects) < self.current_counts['positive']:
            lesion_bin = extra_bins[extra_index % len(extra_bins)]
            pool = self.subjects_by_bin[lesion_bin]
            positive_subjects.append(
                pool[int(rng.integers(0, len(pool)))]
            )
            extra_index += 1
        rng.shuffle(positive_subjects)
        for subject_id in positive_subjects:
            schedule.append({
                'sample_type': 'positive',
                'subject_id': subject_id,
                'start_index': -1,
            })
        for _ in range(self.current_counts['topology_negative']):
            schedule.append({
                'sample_type': 'topology_negative',
                'subject_id': '',
                'start_index': int(rng.integers(0, len(self.topology_pool))),
            })
        for _ in range(self.current_counts['brain_negative']):
            schedule.append({
                'sample_type': 'brain_negative',
                'subject_id': '',
                'start_index': int(rng.integers(0, len(self.brain_pool))),
            })
        order = rng.permutation(len(schedule))
        self.schedule = [schedule[int(index)] for index in order]

    def __len__(self):
        return len(self.schedule)

    def _positive_selection(self, subject_id, rng):
        row = train_row_by_subject[subject_id]
        target, crop_mask, region_map, atlas_enabled = (
            load_selection_arrays(row)
        )
        coordinates = np.argwhere(target > 0)
        center = coordinates[
            int(rng.integers(0, len(coordinates)))
        ].astype(np.int64)
        center += np.asarray([
            int(rng.integers(-12, 13)),
            int(rng.integers(-12, 13)),
            int(rng.integers(-4, 5)),
        ])
        center = np.clip(center, 0, np.asarray(target.shape) - 1)
        return (
            row,
            target,
            crop_mask,
            region_map,
            atlas_enabled,
            center,
            'lesion_jitter',
            '',
        )

    def _topology_negative_selection(self, start_index, rng):
        for attempt in range(len(self.topology_pool)):
            subject_id = self.topology_pool[
                (start_index + attempt) % len(self.topology_pool)
            ]
            row = train_row_by_subject[subject_id]
            target, crop_mask, region_map, atlas_enabled = (
                load_selection_arrays(row)
            )
            labels, relation_by_label = topology_candidate_labels(
                target, region_map
            )
            for relation in [2, 1, 3]:
                relation_labels = [
                    label for label in labels
                    if relation_by_label.get(label) == relation
                ]
                if not relation_labels:
                    continue
                center, count = minimum_overlap_center(
                    target,
                    np.isin(region_map, relation_labels) & crop_mask,
                    rng,
                )
                if center is not None and count == 0:
                    return (
                        row,
                        target,
                        crop_mask,
                        region_map,
                        atlas_enabled,
                        center,
                        'topology_hard_negative',
                        RELATION_NAME[relation],
                    )
        raise RuntimeError('Could not construct a topology hard negative.')

    def _brain_negative_selection(self, start_index, rng):
        for attempt in range(len(self.brain_pool)):
            subject_id = self.brain_pool[
                (start_index + attempt) % len(self.brain_pool)
            ]
            row = train_row_by_subject[subject_id]
            target, crop_mask, region_map, atlas_enabled = (
                load_selection_arrays(row)
            )
            center, count = minimum_overlap_center(
                target, crop_mask, rng
            )
            if center is not None and count == 0:
                return (
                    row,
                    target,
                    crop_mask,
                    region_map,
                    atlas_enabled,
                    center,
                    'brain_hard_negative',
                    '',
                )
        raise RuntimeError('Could not construct a brain hard negative.')

    def __getitem__(self, index):
        entry = self.schedule[int(index)]
        sample_seed = (
            self.base_seed
            + self.epoch * 100_003
            + int(index) * 997
        )
        rng = np.random.default_rng(sample_seed)
        if entry['sample_type'] == 'positive':
            selected = self._positive_selection(entry['subject_id'], rng)
        elif entry['sample_type'] == 'topology_negative':
            selected = self._topology_negative_selection(
                entry['start_index'], rng
            )
        else:
            selected = self._brain_negative_selection(
                entry['start_index'], rng
            )
        (
            row,
            target_binary,
            crop_mask,
            region_map,
            atlas_enabled,
            center,
            center_source,
            topology_relation,
        ) = selected
        subject_id = str(row['subject_id'])
        with np.load(
            Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
        ) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            binary_loss_weight = float(
                np.asarray(data['binary_loss_weight']).item()
            )
        slices = patch_slices_from_center(
            target_binary.shape, center, PATCH_SIZE
        )
        image_patch = extract_patch(
            image, slices, PATCH_SIZE, spatial_axes=(1, 2, 3)
        )
        binary_patch = extract_patch(
            target_binary, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        soft_patch = extract_patch(
            target_soft, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        region_patch = extract_patch(
            region_map, slices, PATCH_SIZE, spatial_axes=(0, 1, 2)
        )
        target_voxels = int(binary_patch.sum())
        if entry['sample_type'] == 'positive' and target_voxels <= 0:
            raise RuntimeError(f'{subject_id}: positive patch lost target.')
        if entry['sample_type'] != 'positive' and target_voxels != 0:
            raise RuntimeError(f'{subject_id}: v4 negative patch has target.')
        image_patch = clinical_window_recalibration(image_patch)
        image_patch = augment_training_image(image_patch, rng)

        graph = load_graph_inputs(row)
        graph = {
            key: np.asarray(value).copy()
            if isinstance(value, np.ndarray)
            else value
            for key, value in graph.items()
        }
        territory_feature_mask = (
            rng.random(graph['territory_features'].shape)
            < GRAPH_FEATURE_MASK_PROBABILITY
        ) & (graph['territory_feature_valid'] > 0)
        vessel_feature_mask = (
            rng.random(graph['vessel_features'].shape)
            < GRAPH_FEATURE_MASK_PROBABILITY
        ) & (graph['vessel_feature_valid'] > 0)
        graph['territory_features'][territory_feature_mask] = 0.0
        graph['territory_feature_valid'][territory_feature_mask] = 0.0
        graph['vessel_features'][vessel_feature_mask] = 0.0
        graph['vessel_feature_valid'][vessel_feature_mask] = 0.0

        territory_target = territory_presence_target(
            target_binary, region_map
        )
        case_loss_weight = (
            positive_case_weight(subject_id)
            if entry['sample_type'] == 'positive'
            else 1.0
        )
        return {
            'subject_id': subject_id,
            'sample_type': entry['sample_type'],
            'center_source': center_source,
            'topology_relation': topology_relation,
            'image': torch.from_numpy(image_patch),
            'region_map': torch.from_numpy(region_patch.astype(np.int64)),
            'target_binary': torch.from_numpy(binary_patch[None].astype(np.float32)),
            'target_soft': torch.from_numpy(soft_patch[None].astype(np.float32)),
            'binary_loss_weight': torch.as_tensor(
                binary_loss_weight, dtype=torch.float32
            ),
            'case_loss_weight': torch.as_tensor(
                case_loss_weight, dtype=torch.float32
            ),
            'territory_target': torch.from_numpy(territory_target),
            'territory_features': torch.from_numpy(graph['territory_features']),
            'territory_feature_valid': torch.from_numpy(
                graph['territory_feature_valid']
            ),
            'territory_valid': torch.from_numpy(graph['territory_valid']),
            'territory_reliability': torch.from_numpy(
                graph['territory_reliability']
            ),
            'vessel_features': torch.from_numpy(graph['vessel_features']),
            'vessel_feature_valid': torch.from_numpy(
                graph['vessel_feature_valid']
            ),
            'vessel_valid': torch.from_numpy(graph['vessel_valid']),
            'case_features': torch.from_numpy(graph['case_features']),
            'atlas_enabled': torch.as_tensor(
                atlas_enabled, dtype=torch.bool
            ),
        }


def to_device(batch):
    return {
        key: value.to(device, non_blocking=True)
        if torch.is_tensor(value)
        else value
        for key, value in batch.items()
    }


def spatially_augment_batch(batch):
    batch_size = int(batch['image'].shape[0])
    apply_mask = (
        torch.rand(batch_size, device=device)
        < float(policy['spatial_affine_probability'])
    )
    if not bool(apply_mask.any()):
        return batch, 0.0
    theta = torch.zeros(
        batch_size, 3, 4, device=device, dtype=torch.float32
    )
    theta[:, 0, 0] = 1.0
    theta[:, 1, 1] = 1.0
    theta[:, 2, 2] = 1.0
    for index in torch.nonzero(apply_mask, as_tuple=False).reshape(-1):
        index = int(index.item())
        angle = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * math.radians(float(policy['spatial_rotation_degrees']))
        )
        scale = torch.empty((), device=device).uniform_(
            float(policy['spatial_scale_range'][0]),
            float(policy['spatial_scale_range'][1]),
        )
        cosine = torch.cos(angle) * scale
        sine = torch.sin(angle) * scale
        
        
        theta[index, 1, 1] = cosine
        theta[index, 1, 2] = -sine
        theta[index, 2, 1] = sine
        theta[index, 2, 2] = cosine
        theta[index, 0, 3] = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * float(policy['spatial_translation_fraction_z'])
        )
        theta[index, 1, 3] = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * float(policy['spatial_translation_fraction_xy'])
        )
        theta[index, 2, 3] = (
            (torch.rand((), device=device) * 2.0 - 1.0)
            * float(policy['spatial_translation_fraction_xy'])
        )
    grid = F.affine_grid(
        theta,
        size=batch['image'].shape,
        align_corners=False,
    )
    transformed = dict(batch)
    intensity = F.grid_sample(
        batch['image'][:, :13],
        grid,
        mode='bilinear',
        padding_mode='zeros',
        align_corners=False,
    )
    validity = F.grid_sample(
        batch['image'][:, 13:],
        grid,
        mode='nearest',
        padding_mode='zeros',
        align_corners=False,
    )
    transformed['image'] = torch.cat([intensity, validity], dim=1)
    transformed['target_soft'] = F.grid_sample(
        batch['target_soft'],
        grid,
        mode='bilinear',
        padding_mode='zeros',
        align_corners=False,
    ).clamp(0.0, 1.0)
    transformed['target_binary'] = F.grid_sample(
        batch['target_binary'],
        grid,
        mode='nearest',
        padding_mode='zeros',
        align_corners=False,
    )
    transformed['region_map'] = F.grid_sample(
        batch['region_map'][:, None].float(),
        grid,
        mode='nearest',
        padding_mode='zeros',
        align_corners=False,
    )[:, 0].long()
    return transformed, float(apply_mask.float().mean().item())


def compute_training_loss(study_model, batch):
    batch, spatial_fraction = spatially_augment_batch(batch)
    model_inputs = {
        key: batch[key]
        for key in [
            'image',
            'region_map',
            'territory_features',
            'territory_feature_valid',
            'territory_valid',
            'territory_reliability',
            'vessel_features',
            'vessel_feature_valid',
            'vessel_valid',
            'case_features',
            'atlas_enabled',
        ]
    }
    output = study_model(**model_inputs)
    main_loss, main_parts = segmentation_loss(
        output['logits'],
        batch['target_soft'],
        batch['target_binary'],
        batch['binary_loss_weight'],
        batch['case_loss_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    half_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_half_logits'].shape[2:],
    )
    half_loss, _ = segmentation_loss(
        output['auxiliary_half_logits'],
        half_soft,
        half_binary,
        batch['binary_loss_weight'],
        batch['case_loss_weight'],
    )
    quarter_soft = F.interpolate(
        batch['target_soft'],
        size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear',
        align_corners=False,
    )
    quarter_binary = F.adaptive_max_pool3d(
        batch['target_binary'],
        output['auxiliary_quarter_logits'].shape[2:],
    )
    quarter_loss, _ = segmentation_loss(
        output['auxiliary_quarter_logits'],
        quarter_soft,
        quarter_binary,
        batch['binary_loss_weight'],
        batch['case_loss_weight'],
    )
    node_mask = (
        batch['territory_valid'].bool()
        & output['effective_atlas_enabled'][:, None]
    )
    node_bce = F.binary_cross_entropy_with_logits(
        output['territory_node_logits'].float(),
        batch['territory_target'].float(),
        reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,),
            float(policy['territory_positive_weight']),
            device=device,
        ),
    )
    node_weights = (
        node_mask.float() * batch['case_loss_weight'][:, None]
    )
    if bool(node_mask.any()):
        node_loss = (node_bce * node_weights).sum() / (
            node_weights.sum().clamp_min(1.0)
        )
    else:
        node_loss = output['territory_node_logits'].sum() * 0.0
    total = (
        main_loss
        + float(policy['deep_supervision_weights']['half_resolution'])
        * half_loss
        + float(policy['deep_supervision_weights']['quarter_resolution'])
        * quarter_loss
        + float(policy['territory_node_loss_weight']) * node_loss
    )
    return total, {
        'total_loss': total,
        'main_loss': main_loss,
        'soft_dice_loss': main_parts['soft_dice'],
        'hard_background_bce_loss': main_parts['hard_background_bce'],
        'hard_focal_tversky_loss': main_parts['hard_focal_tversky'],
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
        'spatially_augmented_fraction': torch.as_tensor(
            spatial_fraction, device=device
        ),
    }


def epoch_loader(dataset, epoch):
    dataset.set_epoch(epoch)
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda',
        persistent_workers=False,
        drop_last=False,
    )


def update_ema(ema_model, study_model, maximum_decay, update_step):
    
    
    decay = min(
        float(maximum_decay),
        float(update_step + 1) / float(update_step + 10),
    )
    with torch.no_grad():
        for ema_parameter, parameter in zip(
            ema_model.parameters(), study_model.parameters()
        ):
            ema_parameter.mul_(decay).add_(parameter, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(
            ema_model.buffers(), study_model.buffers()
        ):
            ema_buffer.copy_(buffer)


def threshold_counts(probability, target):
    probability = np.asarray(probability, dtype=np.float32).reshape(-1)
    target = np.asarray(target, dtype=bool).reshape(-1)
    edges = np.concatenate((
        np.asarray([-np.inf], dtype=np.float32),
        THRESHOLDS.astype(np.float32),
        np.asarray([np.inf], dtype=np.float32),
    ))
    positive_histogram = np.histogram(
        probability[target], bins=edges
    )[0].astype(np.int64)
    negative_histogram = np.histogram(
        probability[~target], bins=edges
    )[0].astype(np.int64)
    true_positive = np.cumsum(positive_histogram[::-1])[::-1][1:]
    false_positive = np.cumsum(negative_histogram[::-1])[::-1][1:]
    truth = int(target.sum())
    predicted = true_positive + false_positive
    denominator = predicted + truth
    dice = np.divide(
        2.0 * true_positive,
        denominator,
        out=np.ones_like(denominator, dtype=np.float64),
        where=denominator > 0,
    )
    precision = np.divide(
        true_positive,
        predicted,
        out=np.where(truth == 0, 1.0, 0.0).astype(np.float64)
        * np.ones_like(predicted, dtype=np.float64),
        where=predicted > 0,
    )
    recall = (
        np.ones_like(true_positive, dtype=np.float64)
        if truth == 0
        else true_positive.astype(np.float64) / truth
    )
    return true_positive, predicted, truth, dice, precision, recall


@torch.no_grad()
def validate_all_subjects(study_model, rows, epoch):
    study_model.eval()
    case_rows = []
    for case_index, row in enumerate(rows, start=1):
        subject_id = str(row['subject_id'])
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
            raise RuntimeError(f'{subject_id}: validation NPZ hash changed.')
        with np.load(npz_path, allow_pickle=False) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            target = np.asarray(data['target_binary'], dtype=np.uint8)
            spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        image = clinical_window_recalibration(image)
        region_map, _ = load_region_map(row, target.shape)
        graph_inputs = load_graph_inputs(row)
        probability = sliding_window_probability(
            study_model, row, image, region_map, graph_inputs
        )
        (
            intersection,
            predicted,
            truth,
            dice,
            precision,
            recall,
        ) = threshold_counts(probability, target > 0)
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        for threshold_index, threshold in enumerate(THRESHOLDS):
            predicted_voxels = int(predicted[threshold_index])
            case_rows.append({
                'epoch': epoch,
                'subject_id': subject_id,
                'lesion_volume_bin': str(
                    row.get('lesion_volume_bin', 'unknown')
                ),
                'threshold': float(threshold),
                'dice': float(dice[threshold_index]),
                'precision': float(precision[threshold_index]),
                'recall': float(recall[threshold_index]),
                'intersection': int(intersection[threshold_index]),
                'predicted_voxels': predicted_voxels,
                'target_voxels': truth,
                'predicted_volume_ml': predicted_voxels * voxel_volume_ml,
                'target_volume_ml': truth * voxel_volume_ml,
                'absolute_volume_error_ml': (
                    abs(predicted_voxels - truth) * voxel_volume_ml
                ),
            })
        if (
            case_index == 1
            or case_index % 5 == 0
            or case_index == len(rows)
        ):
            print(f'  Validated {case_index}/{len(rows)}: {subject_id}')
    case_frame = pd.DataFrame(case_rows)
    summary = (
        case_frame.groupby('threshold')
        .agg(
            mean_subject_dice=('dice', 'mean'),
            median_subject_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_error_ml=(
                'absolute_volume_error_ml', 'mean'
            ),
            total_intersection=('intersection', 'sum'),
            total_predicted_voxels=('predicted_voxels', 'sum'),
            total_target_voxels=('target_voxels', 'sum'),
        )
        .reset_index()
    )
    summary['global_dice'] = (
        2.0 * summary['total_intersection']
        / np.maximum(
            summary['total_predicted_voxels']
            + summary['total_target_voxels'],
            1,
        )
    )
    summary = summary.sort_values(
        ['mean_subject_dice', 'median_subject_dice', 'mean_recall', 'threshold'],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = summary.iloc[0].to_dict()
    selected_threshold = float(selected['threshold'])
    selected_cases = case_frame[
        np.isclose(case_frame['threshold'], selected_threshold)
    ].copy()
    lesion_bin_summary = (
        selected_cases.groupby('lesion_volume_bin', dropna=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_dice=('dice', 'mean'),
            median_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
        )
        .reset_index()
        .sort_values('lesion_volume_bin')
    )
    epoch_directory = VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_directory.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(case_frame, epoch_directory / 'validation_cases.csv')
    atomic_write_csv(summary, epoch_directory / 'threshold_summary.csv')
    atomic_write_csv(
        lesion_bin_summary,
        epoch_directory / 'lesion_bin_summary.csv',
    )
    selected['lesion_bin_mean_dice'] = {
        str(row['lesion_volume_bin']): float(row['mean_dice'])
        for row in lesion_bin_summary.to_dict('records')
    }
    atomic_write_json(
        {
            'epoch': epoch,
            'validation_model': 'ema',
            'selected_threshold': selected_threshold,
            'mean_subject_dice': float(selected['mean_subject_dice']),
            'median_subject_dice': float(selected['median_subject_dice']),
            'global_dice': float(selected['global_dice']),
            'mean_precision': float(selected['mean_precision']),
            'mean_recall': float(selected['mean_recall']),
            'lesion_bin_mean_dice': selected['lesion_bin_mean_dice'],
            'test_targets_read': False,
        },
        epoch_directory / 'validation_summary.json',
    )
    return selected





training_dataset = Trial8V4PatchDataset(train_rows, SEED + 31)
if len(training_dataset) != SAMPLES_PER_EPOCH:
    raise RuntimeError('V4 dataset epoch length changed.')

updates_per_epoch = math.ceil(
    math.ceil(SAMPLES_PER_EPOCH / BATCH_SIZE) / ACCUMULATION_STEPS
)
total_update_steps = MAX_EPOCHS * updates_per_epoch
warmup_steps = 3 * updates_per_epoch


def learning_rate_multiplier(step):
    if step < warmup_steps:
        return float(step + 1) / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(
        total_update_steps - warmup_steps, 1
    )
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


set_determinism(SEED + 37)
training_model = VascuFateNet(
    schema=graph_schema,
    image_channels=15,
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)
if parameter_count(training_model) != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Fresh v4 model parameter count changed.')

ema_model = copy.deepcopy(training_model).to(device).eval()
ema_model.requires_grad_(False)
if parameter_count(ema_model) != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('EMA shadow parameter count changed.')
if parameter_count(ema_model, True) != 0:
    raise RuntimeError('EMA shadow unexpectedly has trainable parameters.')

training_optimizer = torch.optim.AdamW(
    training_model.parameters(),
    lr=BASE_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
training_scheduler = torch.optim.lr_scheduler.LambdaLR(
    training_optimizer, lr_lambda=learning_rate_multiplier
)
training_amp_scaler = torch.amp.GradScaler(
    device.type, enabled=AMP_SCALER_ENABLED
)

start_epoch = 1
global_update_step = 0
best_mean_dice = -1.0
best_epoch = 0
best_threshold = 0.50
validations_without_improvement = 0
history_rows = []

if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'training_version_matches': checkpoint.get('training_version')
        == TRAINING_VERSION,
        'v4_gate_hash_matches': checkpoint.get('v4_gate_sha256')
        == v4_gate_sha256,
        'manifest_hash_matches': checkpoint.get(
            'verified_manifest_sha256'
        ) == verified_manifest_sha256,
        'scaler_hash_matches': checkpoint.get('scaler_sha256')
        == EXPECTED_SCALER_SHA256,
        'schema_hash_matches': checkpoint.get('graph_schema_sha256')
        == safe_schema_sha256,
        'parameter_count_matches': checkpoint.get('total_parameters')
        == EXPECTED_TOTAL_PARAMETERS,
        'test_targets_not_read': checkpoint.get('test_targets_read')
        is False,
        'ema_state_present': 'ema_model_state_dict' in checkpoint,
    }
    if not all(checkpoint_checks.values()):
        failed = [
            name for name, passed in checkpoint_checks.items() if not passed
        ]
        raise RuntimeError('V4 checkpoint mismatch: ' + ', '.join(failed))
    training_model.load_state_dict(checkpoint['model_state_dict'])
    ema_model.load_state_dict(checkpoint['ema_model_state_dict'])
    training_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    training_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    if AMP_SCALER_ENABLED:
        training_amp_scaler.load_state_dict(
            checkpoint['amp_scaler_state_dict']
        )
    restore_rng_state(checkpoint['rng_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    global_update_step = int(checkpoint['global_update_step'])
    best_mean_dice = float(checkpoint['best_mean_dice'])
    best_epoch = int(checkpoint['best_epoch'])
    best_threshold = float(checkpoint['best_threshold'])
    validations_without_improvement = int(
        checkpoint['validations_without_improvement']
    )
    history_rows = list(checkpoint.get('history_rows', []))
    print(
        f'Resuming Trial8 v4 after epoch {start_epoch - 1}; '
        f'best EMA Dice={best_mean_dice:.4f} at epoch {best_epoch}.'
    )
else:
    print('Starting Trial8 v4 from a fresh deterministic initialization.')

training_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'training_version': TRAINING_VERSION,
    'code_revision': CODE_REVISION,
    'training_policy': policy,
    'v4_gate_path': str(V4_GATE_PATH),
    'v4_gate_sha256': v4_gate_sha256,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_sha256': EXPECTED_SCALER_SHA256,
    'graph_schema_sha256': safe_schema_sha256,
    'architecture_sha256': architecture_sha256,
    'total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
    'ema_shadow_parameters': EXPECTED_TOTAL_PARAMETERS,
    'ema_shadow_trainable_parameters': 0,
    'train_subjects': 102,
    'validation_subjects': 23,
    'test_subjects_accessed': 0,
    'mixed_precision_dtype': MIXED_PRECISION_NAME,
    'gradient_scaler_enabled': AMP_SCALER_ENABLED,
    'loss_reduction_dtype': 'float32',
    'focal_tversky_error_floor': V4_TVERSKY_ERROR_FLOOR,
    'ohem_background_to_foreground_ratio': (
        V4_OHEM_BACKGROUND_TO_FOREGROUND_RATIO
    ),
    'ohem_minimum_background_voxels': V4_OHEM_MINIMUM_BACKGROUND_VOXELS,
    'ohem_maximum_background_voxels': V4_OHEM_MAXIMUM_BACKGROUND_VOXELS,
}
if TRAINING_CONFIG_PATH.exists():
    existing = json.loads(
        TRAINING_CONFIG_PATH.read_text(encoding='utf-8')
    )
    immutable = [
        'training_version',
        'training_policy',
        'v4_gate_sha256',
        'verified_manifest_sha256',
        'scaler_sha256',
        'graph_schema_sha256',
        'architecture_sha256',
        'total_parameters',
    ]
    if any(existing.get(key) != training_config.get(key) for key in immutable):
        raise RuntimeError('Existing v4 training configuration changed.')
    existing['code_revision'] = CODE_REVISION
    existing['mixed_precision_dtype'] = MIXED_PRECISION_NAME
    existing['gradient_scaler_enabled'] = AMP_SCALER_ENABLED
    existing['loss_reduction_dtype'] = 'float32'
    existing['focal_tversky_error_floor'] = V4_TVERSKY_ERROR_FLOOR
    existing['ohem_background_to_foreground_ratio'] = (
        V4_OHEM_BACKGROUND_TO_FOREGROUND_RATIO
    )
    existing['ohem_minimum_background_voxels'] = (
        V4_OHEM_MINIMUM_BACKGROUND_VOXELS
    )
    existing['ohem_maximum_background_voxels'] = (
        V4_OHEM_MAXIMUM_BACKGROUND_VOXELS
    )
    atomic_write_json(existing, TRAINING_CONFIG_PATH)
else:
    atomic_write_json(training_config, TRAINING_CONFIG_PATH)

print('\n' + '=' * 80)
print('TRIAL8 V4 SCALE-BALANCED OHEM TRAINING START')
print('=' * 80)
print(f'Device:                             {device}')
if device.type == 'cuda':
    print(f'GPU:                                {torch.cuda.get_device_name(0)}')
print('Train / validation / test access:   102 / 23 / 0')
print(f'Patch size:                         {PATCH_SIZE}')
print(f'Samples per epoch:                  {SAMPLES_PER_EPOCH}')
print('Curriculum phases (pos/topo/brain): 144/30/30 -> 122/41/41 -> 102/51/51')
print(f'Batch / accumulation / effective:   {BATCH_SIZE} / {ACCUMULATION_STEPS} / {BATCH_SIZE * ACCUMULATION_STEPS}')
print(f'Updates per epoch:                  {updates_per_epoch}')
print(f'Maximum epochs:                     {MAX_EPOCHS}')
print(f'Early-stopping patience:            {PATIENCE_VALIDATIONS} epochs')
print(f'Earliest stopping epoch:            {EARLY_STOPPING_MINIMUM_EPOCH}')
print(f'Learning rate / weight decay:       {BASE_LEARNING_RATE:.1e} / {WEIGHT_DECAY:.1e}')
print(f'EMA decay:                          {EMA_DECAY:.3f}')
print(f'Mixed precision:                    {MIXED_PRECISION_NAME}')
print('Loss reductions:                    float32')
print(f'Focal-Tversky FP/FN weights:        {V4_TVERSKY_ALPHA:.2f} / {V4_TVERSKY_BETA:.2f}')
print(f'OHEM background ratio/min/max:      {V4_OHEM_BACKGROUND_TO_FOREGROUND_RATIO:.1f} / {V4_OHEM_MINIMUM_BACKGROUND_VOXELS} / {V4_OHEM_MAXIMUM_BACKGROUND_VOXELS}')
print(f'Total/trainable parameters:         {EXPECTED_TOTAL_PARAMETERS:,} / {EXPECTED_TOTAL_PARAMETERS:,}')
print(f'EMA shadow/trainable parameters:    {EXPECTED_TOTAL_PARAMETERS:,} / 0')
print('Clinical windows:                   CTA/CBF/MTT/Tmax recalibrated')
print('Novel curriculum:                   subject-covered scale curriculum + OHEM')
print(f'Checkpoint directory:               {TRAINING_ROOT}')





training_already_complete = (
    start_epoch > MAX_EPOCHS
    or (
        start_epoch - 1 >= EARLY_STOPPING_MINIMUM_EPOCH
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    )
)
stop_reason = (
    'already_complete_on_resume'
    if training_already_complete
    else 'maximum_epochs_reached'
)
if training_already_complete:
    print('The v4 checkpoint already satisfies a terminal stop condition.')

for epoch in range(
    start_epoch,
    start_epoch if training_already_complete else MAX_EPOCHS + 1,
):
    epoch_start = datetime.now(timezone.utc)
    loader = epoch_loader(training_dataset, epoch)
    expected_curriculum_counts = dict(training_dataset.current_counts)
    curriculum_phase = int(training_dataset.current_phase)
    training_model.train()
    training_optimizer.zero_grad(set_to_none=True)
    running = {
        'total_loss': 0.0,
        'main_loss': 0.0,
        'soft_dice_loss': 0.0,
        'hard_background_bce_loss': 0.0,
        'hard_focal_tversky_loss': 0.0,
        'half_loss': 0.0,
        'quarter_loss': 0.0,
        'node_loss': 0.0,
        'atlas_retained_fraction': 0.0,
        'spatially_augmented_fraction': 0.0,
    }
    processed_batches = 0
    processed_samples = 0
    positive_samples_seen = 0
    topology_samples_seen = 0
    brain_negative_samples_seen = 0
    gradient_norms = []
    amp_skipped_updates = 0
    accumulation_subjects = []
    accumulation_sample_types = []
    for batch_index, cpu_batch in enumerate(loader, start=1):
        batch_samples = int(cpu_batch['target_binary'].shape[0])
        processed_samples += batch_samples
        positive_samples_seen += sum(
            str(value) == 'positive' for value in cpu_batch['sample_type']
        )
        topology_samples_seen += sum(
            str(value) == 'topology_negative'
            for value in cpu_batch['sample_type']
        )
        brain_negative_samples_seen += sum(
            str(value) == 'brain_negative'
            for value in cpu_batch['sample_type']
        )
        accumulation_subjects.extend(
            str(value) for value in cpu_batch['subject_id']
        )
        accumulation_sample_types.extend(
            str(value) for value in cpu_batch['sample_type']
        )
        batch = to_device(cpu_batch)
        with torch.autocast(
            device_type=device.type,
            dtype=MIXED_PRECISION_DTYPE,
            enabled=device.type == 'cuda',
        ):
            total_loss, parts = compute_training_loss(
                training_model, batch
            )
            accumulation_group_start = (
                (batch_index - 1) // ACCUMULATION_STEPS
            ) * ACCUMULATION_STEPS
            accumulation_group_size = min(
                ACCUMULATION_STEPS,
                len(loader) - accumulation_group_start,
            )
            scaled_loss = total_loss / accumulation_group_size
        training_amp_scaler.scale(scaled_loss).backward()
        for name in running:
            running[name] += float(parts[name].detach().cpu())
        processed_batches += 1
        should_update = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(loader)
        )
        if should_update:
            training_amp_scaler.unscale_(training_optimizer)
            gradient_norm = float(
                torch.nn.utils.clip_grad_norm_(
                    training_model.parameters(), GRADIENT_CLIP_NORM
                ).item()
            )
            if not math.isfinite(gradient_norm):
                nonfinite_parameters = [
                    name
                    for name, parameter in training_model.named_parameters()
                    if parameter.grad is not None
                    and not bool(torch.isfinite(parameter.grad).all().item())
                ]
                training_optimizer.zero_grad(set_to_none=True)
                raise RuntimeError(
                    'Non-finite bfloat16/float32 gradient at '
                    f'epoch {epoch}, batch {batch_index}; '
                    f'subjects={accumulation_subjects}; '
                    f'sample_types={accumulation_sample_types}; '
                    f'parameters={nonfinite_parameters[:20]}.'
                )
            gradient_norms.append(gradient_norm)
            training_amp_scaler.step(training_optimizer)
            training_amp_scaler.update()
            training_optimizer.zero_grad(set_to_none=True)
            training_scheduler.step()
            global_update_step += 1
            update_ema(
                ema_model,
                training_model,
                EMA_DECAY,
                global_update_step,
            )
            accumulation_subjects.clear()
            accumulation_sample_types.clear()

    if not gradient_norms:
        raise RuntimeError(
            f'Every optimizer update was skipped at epoch {epoch}.'
        )

    epoch_means = {
        name: value / max(processed_batches, 1)
        for name, value in running.items()
    }
    if not all(math.isfinite(value) for value in epoch_means.values()):
        raise RuntimeError(f'Non-finite v4 loss at epoch {epoch}.')
    curriculum_counts_ok = (
        positive_samples_seen == expected_curriculum_counts['positive']
        and topology_samples_seen
        == expected_curriculum_counts['topology_negative']
        and brain_negative_samples_seen
        == expected_curriculum_counts['brain_negative']
        and processed_samples == SAMPLES_PER_EPOCH
    )
    if not curriculum_counts_ok:
        raise RuntimeError(f'V4 epoch {epoch} curriculum counts changed.')

    print(f'Running full EMA validation at epoch {epoch}...')
    validation_result = validate_all_subjects(
        ema_model, validation_rows, epoch
    )
    current_dice = float(validation_result['mean_subject_dice'])
    current_threshold = float(validation_result['threshold'])
    improved = current_dice > best_mean_dice + IMPROVEMENT_EPSILON
    if improved:
        best_mean_dice = current_dice
        best_epoch = epoch
        best_threshold = current_threshold
        validations_without_improvement = 0
    else:
        validations_without_improvement += 1

    epoch_seconds = (
        datetime.now(timezone.utc) - epoch_start
    ).total_seconds()
    history_row = {
        'epoch': epoch,
        'global_update_step': global_update_step,
        'learning_rate': float(training_optimizer.param_groups[0]['lr']),
        **epoch_means,
        'mean_gradient_norm': float(np.mean(gradient_norms)),
        'maximum_gradient_norm': float(np.max(gradient_norms)),
        'amp_skipped_updates': 0,
        'mixed_precision_dtype': MIXED_PRECISION_NAME,
        'curriculum_phase': curriculum_phase,
        'expected_positive_samples': expected_curriculum_counts['positive'],
        'expected_topology_negative_samples': expected_curriculum_counts[
            'topology_negative'
        ],
        'expected_brain_negative_samples': expected_curriculum_counts[
            'brain_negative'
        ],
        'positive_patch_fraction': (
            positive_samples_seen / processed_samples
        ),
        'topology_negative_fraction': (
            topology_samples_seen / processed_samples
        ),
        'brain_negative_fraction': (
            brain_negative_samples_seen / processed_samples
        ),
        'validation_model': 'ema',
        'validation_threshold': current_threshold,
        'validation_mean_subject_dice': current_dice,
        'validation_median_subject_dice': float(
            validation_result['median_subject_dice']
        ),
        'validation_global_dice': float(validation_result['global_dice']),
        'validation_mean_precision': float(
            validation_result['mean_precision']
        ),
        'validation_mean_recall': float(
            validation_result['mean_recall']
        ),
        'validation_mean_absolute_volume_error_ml': float(
            validation_result['mean_absolute_volume_error_ml']
        ),
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'improved': improved,
        'epoch_seconds': epoch_seconds,
        'test_targets_read': False,
    }
    for lesion_bin, value in validation_result[
        'lesion_bin_mean_dice'
    ].items():
        history_row[f'validation_mean_dice_{lesion_bin}'] = float(value)
    history_rows.append(to_builtin(history_row))
    atomic_write_csv(pd.DataFrame(history_rows), HISTORY_PATH)

    checkpoint_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'epoch': epoch,
        'global_update_step': global_update_step,
        'model_state_dict': training_model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': training_optimizer.state_dict(),
        'scheduler_state_dict': training_scheduler.state_dict(),
        'amp_scaler_state_dict': training_amp_scaler.state_dict(),
        'rng_state': capture_rng_state(),
        'best_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'history_rows': history_rows,
        'code_revision': CODE_REVISION,
        'amp_skipped_updates_this_epoch': 0,
        'mixed_precision_dtype': MIXED_PRECISION_NAME,
        'v4_gate_sha256': v4_gate_sha256,
        'verified_manifest_sha256': verified_manifest_sha256,
        'scaler_sha256': EXPECTED_SCALER_SHA256,
        'graph_schema_sha256': safe_schema_sha256,
        'architecture_sha256': architecture_sha256,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'test_targets_read': False,
    }
    atomic_torch_save(checkpoint_record, LAST_CHECKPOINT_PATH)
    if improved:
        atomic_torch_save(checkpoint_record, BEST_CHECKPOINT_PATH)

    print(
        f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
        f'loss={epoch_means["total_loss"]:.4f}, '
        f'main={epoch_means["main_loss"]:.4f}, '
        f'node={epoch_means["node_loss"]:.4f}, '
        f'lr={history_row["learning_rate"]:.2e}, '
        f'grad={history_row["mean_gradient_norm"]:.3f}, '
        'amp-skips=0, '
        f'phase={curriculum_phase}, '
        f'positive={history_row["positive_patch_fraction"]:.3f}, '
        f'topology-neg={history_row["topology_negative_fraction"]:.3f}, '
        f'brain-neg={history_row["brain_negative_fraction"]:.3f}, '
        f'time={epoch_seconds / 60.0:.1f} min'
    )
    print(
        f'  EMA validation: mean Dice={current_dice:.4f}, '
        f'median Dice={validation_result["median_subject_dice"]:.4f}, '
        f'global Dice={validation_result["global_dice"]:.4f}, '
        f'precision={validation_result["mean_precision"]:.4f}, '
        f'recall={validation_result["mean_recall"]:.4f}, '
        f'threshold={current_threshold:.3f}, '
        f'best={best_mean_dice:.4f} at epoch {best_epoch}'
    )
    print(
        '  Mean Dice by lesion bin: '
        + ', '.join(
            f'{name}={value:.4f}'
            for name, value in validation_result[
                'lesion_bin_mean_dice'
            ].items()
        )
    )

    atomic_write_json(
        {
            'updated_at_utc': datetime.now(timezone.utc).isoformat(),
            'training_version': TRAINING_VERSION,
            'status': 'running',
            'last_completed_epoch': epoch,
            'global_update_step': global_update_step,
            'best_mean_subject_dice': best_mean_dice,
            'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'validations_without_improvement': (
                validations_without_improvement
            ),
            'total_parameters': EXPECTED_TOTAL_PARAMETERS,
            'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
            'ema_shadow_trainable_parameters': 0,
            'test_targets_read': False,
            'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
            'best_checkpoint_path': (
                str(BEST_CHECKPOINT_PATH)
                if BEST_CHECKPOINT_PATH.exists()
                else ''
            ),
        },
        TRAINING_STATE_PATH,
    )

    if (
        epoch >= EARLY_STOPPING_MINIMUM_EPOCH
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    ):
        stop_reason = 'early_stopping'
        print(
            f'Early stopping after {validations_without_improvement} '
            'full EMA validations without improvement.'
        )
        break

completed_epoch = history_rows[-1]['epoch'] if history_rows else start_epoch - 1
atomic_write_json(
    {
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_version': TRAINING_VERSION,
        'status': 'complete',
        'stop_reason': stop_reason,
        'last_completed_epoch': completed_epoch,
        'global_update_step': global_update_step,
        'best_mean_subject_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'total_parameters': EXPECTED_TOTAL_PARAMETERS,
        'trainable_parameters': EXPECTED_TOTAL_PARAMETERS,
        'ema_shadow_trainable_parameters': 0,
        'train_subjects': 102,
        'validation_subjects': 23,
        'test_targets_read': False,
        'last_checkpoint_path': str(LAST_CHECKPOINT_PATH),
        'best_checkpoint_path': str(BEST_CHECKPOINT_PATH),
        'history_path': str(HISTORY_PATH),
    },
    TRAINING_STATE_PATH,
)

print('\n' + '=' * 80)
print('TRIAL8 V4 SCALE-BALANCED OHEM TRAINING COMPLETE')
print('=' * 80)
print(f'Stop reason:                        {stop_reason}')
print(f'Last completed epoch:               {completed_epoch}')
print(f'Best EMA validation mean Dice:      {best_mean_dice:.4f}')
print(f'Best epoch:                         {best_epoch}')
print(f'Frozen validation threshold:        {best_threshold:.3f}')
print(f'Total/trainable parameters:         {EXPECTED_TOTAL_PARAMETERS:,} / {EXPECTED_TOTAL_PARAMETERS:,}')
print(f'EMA shadow/trainable parameters:    {EXPECTED_TOTAL_PARAMETERS:,} / 0')
print('Test targets read:                  False')
print('Validation cadence:                 every epoch')
print('Novel curriculum:                   subject-covered scale curriculum + OHEM')
print('Scientific core changed:            False')
print(f'Best checkpoint:                    {BEST_CHECKPOINT_PATH}')
print(f'Last checkpoint:                    {LAST_CHECKPOINT_PATH}')
print(f'Training history:                   {HISTORY_PATH}')
print(f'Training state:                     {TRAINING_STATE_PATH}')


In [ ]:
import sys
import os
import gc
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
PREFLIGHT_VERSION = 'v1_train_only_scaler_ragatv2_film'
TRAINING_PREFLIGHT_VERSION = 'v5_official_metric_audit_scaffold'
CODE_REVISION = 'r1_v2_v3_v4_official_metrics_graph_counterfactual'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
MODEL_PREFLIGHT_GATE_PATH = MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / TRAINING_PREFLIGHT_VERSION
)
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'training_preflight_gate.json'
TARGET_FREE_RECHECK_GATE_PATH = OUTPUT_ROOT / 'target_free_recheck_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'training_preflight_checks.csv'
TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_validation_target_audit.csv'
PATCH_AUDIT_PATH = OUTPUT_ROOT / 'lesion_patch_sampling_audit.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'real_loss_components.json'
VALIDATION_AUDIT_PATH = OUTPUT_ROOT / 'validation_smoke_metrics.csv'

SEED = 20260919
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_052_628
EXPECTED_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'training_preflight_version': TRAINING_PREFLIGHT_VERSION,
        'training_preflight_passed': False,
        'full_training_authorized': False,
        'status': 'running_or_incomplete',
        'test_targets_read': False,
        'study_training_epochs_completed': 0,
    },
    PREFLIGHT_GATE_PATH,
)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    SCALER_NPZ_PATH,
    SCALER_JSON_PATH,
    ARCHITECTURE_JSON_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_preflight_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')
if model_preflight_gate.get('preflight_passed') is not True:
    raise RuntimeError('Frozen model/scaler preflight did not pass.')
if model_preflight_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Frozen model/scaler preflight does not authorize training.')
if model_preflight_gate.get('total_parameters') != EXPECTED_TOTAL_PARAMETERS:
    raise RuntimeError('Frozen model parameter count changed.')
if model_preflight_gate.get('scaler_sha256') != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen preflight gate scaler hash changed.')
if file_sha256(SCALER_NPZ_PATH) != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen train-only scaler file hash changed.')
if model_preflight_gate.get('architecture_config_sha256') != file_sha256(
    ARCHITECTURE_JSON_PATH
):
    raise RuntimeError('Frozen architecture file hash changed.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Target arrays read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

recomputed_scaler_arrays = {
    'territory_feature_names': np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    'territory_mean': territory_mean,
    'territory_std': territory_std,
    'territory_count': territory_count,
    'vessel_feature_names': np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    'vessel_source_indices': np.asarray(vessel_source_indices, dtype=np.int16),
    'vessel_mean': vessel_mean,
    'vessel_std': vessel_std,
    'vessel_count': vessel_count,
    'case_feature_names': np.asarray(MODEL_CASE_FEATURE_NAMES),
    'case_source_indices': np.asarray(case_source_indices, dtype=np.int16),
    'case_scale_mask': case_scale_mask.astype(np.uint8),
    'case_mean': case_mean,
    'case_std': case_std,
    'case_count': case_count,
    'clip_value': np.asarray(SCALER_CLIP, dtype=np.float32),
    'fit_split': np.asarray('train'),
    'fit_subject_count': np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    'fit_subject_hash': np.asarray(scaler_fit_subject_hash),
    'target_arrays_read': np.asarray(False, dtype=np.uint8),
}
with np.load(SCALER_NPZ_PATH, allow_pickle=False) as frozen_scaler:
    if set(frozen_scaler.files) != set(recomputed_scaler_arrays):
        raise RuntimeError('Frozen scaler array schema changed.')
    for name, expected in recomputed_scaler_arrays.items():
        observed = np.asarray(frozen_scaler[name])
        if observed.dtype.kind in {'f', 'c'}:
            equal = np.array_equal(observed, expected)
        else:
            equal = np.array_equal(observed, expected)
        if not equal:
            raise RuntimeError(f'Frozen scaler value mismatch: {name}')
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)
if scaler_sha256 != EXPECTED_SCALER_SHA256:
    raise RuntimeError('Frozen scaler hash changed after value verification.')

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class VascuFateNet(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        bottleneck = self.global_film(bottleneck, global_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')
preflight_row = preflight_candidates.iloc[0]
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    if not {'image', 'subject_id'}.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNet(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
backbone_parameters = (
    total_parameters - graph_parameters - film_parameters - head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    normal_output = model(**batch, force_atlas_dropout=False)
    dropped_output = model(**batch, force_atlas_dropout=True)

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        dropped_output['atlas_dropped'].item()
        and not dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'], dropped_output['logits'], atol=0.0, rtol=0.0
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
})

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=autocast_enabled,
):
    gradient_output = model(**batch, force_atlas_dropout=False)
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = json.loads(
    ARCHITECTURE_JSON_PATH.read_text(encoding='utf-8')
)
frozen_architecture_checks = {
    'frozen_architecture_model_matches': architecture_config.get('model_version')
    == MODEL_VERSION,
    'frozen_architecture_parameter_count_matches': architecture_config.get(
        'total_parameters'
    )
    == EXPECTED_TOTAL_PARAMETERS,
    'frozen_architecture_scaler_matches': architecture_config.get('scaler_sha256')
    == EXPECTED_SCALER_SHA256,
    'frozen_architecture_manifest_matches': architecture_config.get(
        'verified_manifest_sha256'
    )
    == verified_manifest_sha256,
}
if not all(frozen_architecture_checks.values()):
    failed = [
        name for name, passed in frozen_architecture_checks.items() if not passed
    ]
    raise RuntimeError('Frozen architecture checks failed: ' + ', '.join(failed))
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **frozen_architecture_checks,
    **forward_checks,
    **gradient_checks,
    'parameter_count_in_efficiency_target': 12_000_000
    <= total_parameters
    <= 20_000_000,
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'checkpoint_audit_prerequisites_passed': preflight_passed,
    'model_training_authorized': False,
}
atomic_write_json(gate_record, TARGET_FREE_RECHECK_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 MODEL AND SCALER PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET MODEL AND TRAIN-ONLY SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('Scaler fitted on:                   train only (102 subjects)')
print('Target arrays read:                 False')
print('Checkpoint audit prerequisites:     passed')
print('Model training authorized:          False (diagnostic-only cell)')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Target-free recheck gate:           {TARGET_FREE_RECHECK_GATE_PATH}')
print('\nNext: run the official-metric checkpoint audit below.')









for disposable_name in [
    'model', 'optimizer', 'amp_scaler', 'real_output', 'real_batch',
    'real_soft_tensor', 'real_binary_tensor',
    'real_territory_target_tensor', 'normal_output', 'dropped_output',
    'gradient_output', 'batch', 'smoke_loss', 'total_real_loss',
    'main_loss', 'half_loss', 'quarter_loss', 'node_loss', 'node_bce',
    'before_step', 'validation_probability', 'validation_image',
    'validation_target', 'validation_region', 'validation_graph',
]:
    globals().pop(disposable_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ensure_package('scipy')
from scipy import ndimage
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching
from scipy.stats import rankdata

AUDIT_VERSION = 'v1_official_metrics_v2_v3_v4_graph_counterfactual'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
AUDIT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / AUDIT_VERSION
)
PREDICTION_ROOT = AUDIT_ROOT / 'validation_predictions'
CASE_METRICS_PATH = AUDIT_ROOT / 'validation_case_metrics.csv'
MODEL_SUMMARY_PATH = AUDIT_ROOT / 'model_summary.csv'
LESION_BIN_SUMMARY_PATH = AUDIT_ROOT / 'lesion_bin_summary.csv'
ATLAS_MODE_SUMMARY_PATH = AUDIT_ROOT / 'atlas_mode_summary.csv'
LOCAL_RANK_PATH = AUDIT_ROOT / 'local_rank_then_aggregate.csv'
GRAPH_COUNTERFACTUAL_PATH = AUDIT_ROOT / 'v2_graph_counterfactual.csv'
WORST_CASES_PATH = AUDIT_ROOT / 'worst_cases.csv'
AUDIT_GATE_PATH = AUDIT_ROOT / 'official_metric_audit_gate.json'

assert_project_path_is_safe(AUDIT_ROOT)
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)

IOU_MATCH_THRESHOLD = 0.20
COMPONENT_CONNECTIVITY = 26
MODEL_SPECS = [
    {
        'model_label': 'v2_best',
        'checkpoint_path': (
            CHECKPOINT_ROOT
            / 'v2_regularized_balanced_epochwise'
            / 'best_checkpoint.pt'
        ),
        'state_key': 'model_state_dict',
        'preprocessing': 'frozen_r2_channels',
        'patch_size': (128, 128, 64),
        'amp_dtype': 'float16',
        'force_atlas_dropout': False,
        'counterfactual_reference': '',
    },
    {
        'model_label': 'v2_graph_disabled',
        'checkpoint_path': (
            CHECKPOINT_ROOT
            / 'v2_regularized_balanced_epochwise'
            / 'best_checkpoint.pt'
        ),
        'state_key': 'model_state_dict',
        'preprocessing': 'frozen_r2_channels',
        'patch_size': (128, 128, 64),
        'amp_dtype': 'float16',
        'force_atlas_dropout': True,
        'counterfactual_reference': 'v2_best',
    },
    {
        'model_label': 'v3_ema_best',
        'checkpoint_path': (
            CHECKPOINT_ROOT
            / 'v3_topology_curriculum_clinical_windows_ema'
            / 'best_checkpoint.pt'
        ),
        'state_key': 'ema_model_state_dict',
        'preprocessing': 'clinical_windows_v3_v4',
        'patch_size': (112, 112, 48),
        'amp_dtype': 'bfloat16',
        'force_atlas_dropout': False,
        'counterfactual_reference': '',
    },
    {
        'model_label': 'v4_ema_best',
        'checkpoint_path': (
            CHECKPOINT_ROOT
            / 'v4_scale_balanced_ohem_curriculum_bf16'
            / 'best_checkpoint.pt'
        ),
        'state_key': 'ema_model_state_dict',
        'preprocessing': 'clinical_windows_v3_v4',
        'patch_size': (112, 112, 48),
        'amp_dtype': 'bfloat16',
        'force_atlas_dropout': False,
        'counterfactual_reference': '',
    },
]


def clinical_window_recalibration(image):
    output = np.asarray(image, dtype=np.float32).copy()
    
    
    output[2] = np.clip((output[2] * 1100.0 - 100.0) / 90.0, 0.0, 1.0)
    output[3] = np.clip((output[3] * 150.0) / 35.0, 0.0, 1.0)
    output[5] = np.clip((output[5] * 30.0) / 20.0, 0.0, 1.0)
    output[6] = np.clip((output[6] * 20.0) / 7.0, 0.0, 1.0)
    return output


def preprocess_for_spec(image, preprocessing):
    image = np.asarray(image, dtype=np.float32)
    if preprocessing == 'frozen_r2_channels':
        return image
    if preprocessing == 'clinical_windows_v3_v4':
        return clinical_window_recalibration(image)
    raise RuntimeError(f'Unknown audit preprocessing: {preprocessing}')


def load_graph_inputs(row):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains a target.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(
            data['vessel_features'], dtype=np.float32
        )
        vessel_valid_source = np.asarray(
            data['vessel_feature_valid']
        ) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
    )
    vessel_selected = vessel_source[:, vessel_source_indices]
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_source_indices]
        & vessel_presence[:, None]
    )
    vessel_scaled = masked_scale(
        vessel_selected,
        vessel_feature_valid,
        vessel_mean[None],
        vessel_std[None],
    )
    case_selected = case_source[case_source_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scaled[case_scale_mask] = np.clip(
        (case_scaled[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    return {
        'territory_features': territory_scaled,
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid,
        'territory_reliability': node_reliability,
        'vessel_features': vessel_scaled,
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence,
        'case_features': case_scaled,
    }


def load_region_map(row, spatial_shape):
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if not atlas_enabled:
        return np.zeros(spatial_shape, dtype=np.int64), False
    atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
    if not atlas_path.exists():
        raise FileNotFoundError(f'Missing selected atlas: {atlas_path}')
    atlas_image = nib.load(str(atlas_path))
    labels = np.asanyarray(atlas_image.dataobj)
    if labels.ndim == 4 and labels.shape[-1] == 1:
        labels = labels[..., 0]
    labels = np.rint(labels).astype(np.int64)
    if labels.shape != tuple(spatial_shape):
        raise RuntimeError(f'{row["subject_id"]}: atlas shape mismatch.')
    if labels.min() < 0 or labels.max() > NUM_TERRITORY_NODES:
        raise RuntimeError(
            f'{row["subject_id"]}: atlas labels outside 0..30.'
        )
    return labels, True


def sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(int(round(patch * (1.0 - overlap))), 1)
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def gaussian_importance_map(patch_size):
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in patch_size
    ]
    grid = np.meshgrid(*coordinates, indexing='ij')
    squared_distance = sum(axis ** 2 for axis in grid)
    weights = np.exp(-0.5 * squared_distance / (0.35 ** 2))
    return np.clip(weights, 1e-3, None).astype(np.float32)


def build_study_model():
    study_model = VascuFateNet(
        schema=graph_schema,
        image_channels=15,
        territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
        vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
        case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
        graph_hidden=GRAPH_HIDDEN,
        graph_heads=GRAPH_HEADS,
        graph_layers=GRAPH_LAYERS,
        relation_bases=RELATION_BASES,
        graph_dropout=GRAPH_DROPOUT,
        atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
        film_bound=FILM_BOUND,
    ).to(device)
    if parameter_count(study_model) != EXPECTED_TOTAL_PARAMETERS:
        raise RuntimeError('Audit model parameter count changed.')
    return study_model


def load_read_only_checkpoint(spec):
    path = Path(spec['checkpoint_path'])
    if not path.exists():
        raise FileNotFoundError(f'Required best checkpoint is missing: {path}')
    checkpoint_sha256 = file_sha256(path)
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    checks = {
        'requested_state_present': spec['state_key'] in checkpoint,
        'manifest_hash_matches': checkpoint.get(
            'verified_manifest_sha256'
        ) == EXPECTED_VERIFIED_MANIFEST_SHA256,
        'scaler_hash_matches': checkpoint.get(
            'scaler_sha256'
        ) == EXPECTED_SCALER_SHA256,
        'graph_schema_hash_matches': checkpoint.get(
            'graph_schema_sha256'
        ) == EXPECTED_SAFE_SCHEMA_SHA256,
        'parameter_count_matches': checkpoint.get(
            'total_parameters'
        ) == EXPECTED_TOTAL_PARAMETERS,
        'test_targets_were_not_read': checkpoint.get(
            'test_targets_read'
        ) is False,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'{spec["model_label"]} checkpoint integrity failed: '
            + ', '.join(failed)
        )
    study_model = build_study_model()
    study_model.load_state_dict(checkpoint[spec['state_key']], strict=True)
    study_model.eval()
    metadata = {
        **spec,
        'checkpoint_path': str(path),
        'checkpoint_sha256': checkpoint_sha256,
        'training_version': str(checkpoint.get('training_version', '')),
        'checkpoint_epoch': int(checkpoint['epoch']),
        'best_epoch': int(checkpoint.get('best_epoch', checkpoint['epoch'])),
        'threshold': float(checkpoint.get('best_threshold', 0.50)),
        'reported_best_mean_dice': float(
            checkpoint.get('best_mean_dice', np.nan)
        ),
    }
    return study_model, metadata


def audit_sliding_window_probability(
    study_model,
    row,
    image,
    region_map,
    graph_inputs,
    patch_size,
    amp_dtype,
    force_atlas_dropout,
):
    patch_size = tuple(int(value) for value in patch_size)
    original_shape = tuple(int(value) for value in image.shape[1:])
    padded_shape = tuple(
        max(size, patch) for size, patch in zip(original_shape, patch_size)
    )
    image_padded = np.zeros(
        (image.shape[0], *padded_shape), dtype=np.float32
    )
    region_padded = np.zeros(padded_shape, dtype=np.int64)
    original_slices = tuple(slice(0, size) for size in original_shape)
    image_padded[(slice(None), *original_slices)] = image
    region_padded[original_slices] = region_map
    probability_sum = np.zeros(padded_shape, dtype=np.float32)
    weight_sum = np.zeros(padded_shape, dtype=np.float32)
    importance = gaussian_importance_map(patch_size)
    starts = [
        sliding_starts(size, patch, overlap=0.5)
        for size, patch in zip(padded_shape, patch_size)
    ]
    locations = [
        (x, y, z)
        for x in starts[0]
        for y in starts[1]
        for z in starts[2]
    ]
    torch_dtype = (
        torch.bfloat16 if amp_dtype == 'bfloat16' else torch.float16
    )
    for location_index in range(0, len(locations), 2):
        location_batch = locations[location_index:location_index + 2]
        image_patches = []
        region_patches = []
        for x, y, z in location_batch:
            slices = (
                slice(x, x + patch_size[0]),
                slice(y, y + patch_size[1]),
                slice(z, z + patch_size[2]),
            )
            image_patches.append(image_padded[(slice(None), *slices)])
            region_patches.append(region_padded[slices])
        count = len(location_batch)
        inference_batch = {
            'image': torch.from_numpy(np.stack(image_patches)).to(device),
            'region_map': torch.from_numpy(np.stack(region_patches)).to(device),
            'territory_features': torch.from_numpy(np.repeat(
                graph_inputs['territory_features'][None], count, axis=0
            )).to(device),
            'territory_feature_valid': torch.from_numpy(np.repeat(
                graph_inputs['territory_feature_valid'][None], count, axis=0
            )).to(device),
            'territory_valid': torch.from_numpy(np.repeat(
                graph_inputs['territory_valid'][None], count, axis=0
            )).to(device),
            'territory_reliability': torch.from_numpy(np.repeat(
                graph_inputs['territory_reliability'][None], count, axis=0
            )).to(device),
            'vessel_features': torch.from_numpy(np.repeat(
                graph_inputs['vessel_features'][None], count, axis=0
            )).to(device),
            'vessel_feature_valid': torch.from_numpy(np.repeat(
                graph_inputs['vessel_feature_valid'][None], count, axis=0
            )).to(device),
            'vessel_valid': torch.from_numpy(np.repeat(
                graph_inputs['vessel_valid'][None], count, axis=0
            )).to(device),
            'case_features': torch.from_numpy(np.repeat(
                graph_inputs['case_features'][None], count, axis=0
            )).to(device),
            'atlas_enabled': torch.full(
                (count,),
                as_bool(row['trial8_atlas_enabled']),
                dtype=torch.bool,
                device=device,
            ),
        }
        with torch.no_grad(), torch.autocast(
            device_type=device.type,
            dtype=torch_dtype,
            enabled=device.type == 'cuda',
        ):
            logits = study_model(
                **inference_batch,
                force_atlas_dropout=force_atlas_dropout,
            )['logits']
        probabilities = torch.sigmoid(logits).float().cpu().numpy()[:, 0]
        for probability, (x, y, z) in zip(probabilities, location_batch):
            slices = (
                slice(x, x + patch_size[0]),
                slice(y, y + patch_size[1]),
                slice(z, z + patch_size[2]),
            )
            probability_sum[slices] += probability * importance
            weight_sum[slices] += importance
        del inference_batch, logits, probabilities
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    probability = probability[original_slices]
    if not np.isfinite(probability).all():
        raise RuntimeError(f'{row["subject_id"]}: non-finite probability.')
    if float(probability.min()) < 0.0 or float(probability.max()) > 1.0:
        raise RuntimeError(f'{row["subject_id"]}: probability outside [0, 1].')
    return probability.astype(np.float32, copy=False)


def component_labels(mask):
    structure = np.ones((3, 3, 3), dtype=bool)
    labels, count = ndimage.label(np.asarray(mask, dtype=bool), structure)
    return labels.astype(np.int32, copy=False), int(count)


def lesion_instance_metrics(prediction, target):
    target_labels, target_count = component_labels(target)
    prediction_labels, prediction_count = component_labels(prediction)
    if target_count == 0 and prediction_count == 0:
        return {
            'target_lesions': 0,
            'predicted_lesions': 0,
            'matched_lesions': 0,
            'lesion_false_positives': 0,
            'lesion_false_negatives': 0,
            'lesion_f1': 1.0,
            'absolute_lesion_count_difference': 0,
        }
    if target_count == 0 or prediction_count == 0:
        matched = 0
    else:
        target_sizes = np.bincount(
            target_labels.reshape(-1), minlength=target_count + 1
        )[1:].astype(np.int64)
        prediction_sizes = np.bincount(
            prediction_labels.reshape(-1),
            minlength=prediction_count + 1,
        )[1:].astype(np.int64)
        overlap = (target_labels > 0) & (prediction_labels > 0)
        if bool(overlap.any()):
            pair_index = (
                target_labels[overlap].astype(np.int64)
                * (prediction_count + 1)
                + prediction_labels[overlap].astype(np.int64)
            )
            intersections = np.bincount(
                pair_index,
                minlength=(target_count + 1) * (prediction_count + 1),
            ).reshape(target_count + 1, prediction_count + 1)[1:, 1:]
            unions = (
                target_sizes[:, None]
                + prediction_sizes[None, :]
                - intersections
            )
            iou = np.divide(
                intersections,
                unions,
                out=np.zeros_like(intersections, dtype=np.float64),
                where=unions > 0,
            )
            eligible = csr_matrix(iou >= IOU_MATCH_THRESHOLD)
            matching = maximum_bipartite_matching(
                eligible, perm_type='column'
            )
            matched = int((matching >= 0).sum())
        else:
            matched = 0
    lesion_fp = prediction_count - matched
    lesion_fn = target_count - matched
    denominator = 2 * matched + lesion_fp + lesion_fn
    lesion_f1 = 1.0 if denominator == 0 else 2.0 * matched / denominator
    return {
        'target_lesions': target_count,
        'predicted_lesions': prediction_count,
        'matched_lesions': matched,
        'lesion_false_positives': lesion_fp,
        'lesion_false_negatives': lesion_fn,
        'lesion_f1': float(lesion_f1),
        'absolute_lesion_count_difference': abs(
            prediction_count - target_count
        ),
    }


def official_case_metrics(prediction, target, spacing):
    prediction = np.asarray(prediction, dtype=bool)
    target = np.asarray(target, dtype=bool)
    if prediction.shape != target.shape:
        raise RuntimeError('Prediction and target shapes differ.')
    intersection = int((prediction & target).sum())
    predicted_voxels = int(prediction.sum())
    target_voxels = int(target.sum())
    denominator = predicted_voxels + target_voxels
    dice = 1.0 if denominator == 0 else 2.0 * intersection / denominator
    precision = (
        1.0 if target_voxels == 0 else 0.0
    ) if predicted_voxels == 0 else intersection / predicted_voxels
    recall = 1.0 if target_voxels == 0 else intersection / target_voxels
    voxel_volume_ml = float(np.prod(np.asarray(spacing, dtype=np.float64)) / 1000.0)
    instance = lesion_instance_metrics(prediction, target)
    return {
        'dice': float(dice),
        'precision': float(precision),
        'recall': float(recall),
        'intersection': intersection,
        'predicted_voxels': predicted_voxels,
        'target_voxels': target_voxels,
        'predicted_volume_ml': predicted_voxels * voxel_volume_ml,
        'target_volume_ml': target_voxels * voxel_volume_ml,
        'absolute_volume_difference_ml': (
            abs(predicted_voxels - target_voxels) * voxel_volume_ml
        ),
        **instance,
    }


def prediction_cache_path(model_label, subject_id):
    return PREDICTION_ROOT / model_label / f'{subject_id}_prediction.npz'


def cache_expectation(spec, metadata, row, spatial_shape):
    return {
        'model_label': str(spec['model_label']),
        'subject_id': str(row['subject_id']),
        'checkpoint_sha256': str(metadata['checkpoint_sha256']),
        'source_npz_sha256': str(row['trial8_npz_sha256']),
        'threshold': float(metadata['threshold']),
        'preprocessing': str(spec['preprocessing']),
        'state_key': str(spec['state_key']),
        'patch_size': tuple(int(value) for value in spec['patch_size']),
        'force_atlas_dropout': bool(spec['force_atlas_dropout']),
        'spatial_shape': tuple(int(value) for value in spatial_shape),
    }


def load_cached_prediction(path, expectation):
    if not path.exists():
        return None
    try:
        with np.load(path, allow_pickle=False) as data:
            required = {
                'prediction', 'model_label', 'subject_id',
                'checkpoint_sha256', 'source_npz_sha256', 'threshold',
                'preprocessing', 'state_key', 'patch_size',
                'force_atlas_dropout', 'spatial_shape',
                'pair_probability_mae', 'pair_probability_maximum_error',
            }
            if not required.issubset(data.files):
                return None
            if any('target' in name.lower() for name in data.files):
                return None
            observed = {
                'model_label': scalar_string(data['model_label']),
                'subject_id': scalar_string(data['subject_id']),
                'checkpoint_sha256': scalar_string(data['checkpoint_sha256']),
                'source_npz_sha256': scalar_string(data['source_npz_sha256']),
                'threshold': float(np.asarray(data['threshold']).item()),
                'preprocessing': scalar_string(data['preprocessing']),
                'state_key': scalar_string(data['state_key']),
                'patch_size': tuple(
                    int(value) for value in np.asarray(data['patch_size'])
                ),
                'force_atlas_dropout': bool(
                    np.asarray(data['force_atlas_dropout']).item()
                ),
                'spatial_shape': tuple(
                    int(value) for value in np.asarray(data['spatial_shape'])
                ),
            }
            exact_fields = [
                'model_label', 'subject_id', 'checkpoint_sha256',
                'source_npz_sha256', 'preprocessing', 'state_key',
                'patch_size', 'force_atlas_dropout', 'spatial_shape',
            ]
            if any(observed[name] != expectation[name] for name in exact_fields):
                return None
            if not np.isclose(observed['threshold'], expectation['threshold']):
                return None
            prediction = np.asarray(data['prediction']) > 0
            if prediction.shape != expectation['spatial_shape']:
                return None
            pair_mae = float(np.asarray(data['pair_probability_mae']).item())
            pair_max = float(
                np.asarray(data['pair_probability_maximum_error']).item()
            )
        return prediction, pair_mae, pair_max
    except (OSError, ValueError, KeyError, TypeError):
        return None


def save_prediction_cache(
    path,
    prediction,
    expectation,
    pair_probability_mae=np.nan,
    pair_probability_maximum_error=np.nan,
):
    path.parent.mkdir(parents=True, exist_ok=True)
    atomic_write_npz(
        path,
        prediction=np.asarray(prediction, dtype=np.uint8),
        model_label=np.asarray(expectation['model_label']),
        subject_id=np.asarray(expectation['subject_id']),
        checkpoint_sha256=np.asarray(expectation['checkpoint_sha256']),
        source_npz_sha256=np.asarray(expectation['source_npz_sha256']),
        threshold=np.asarray(expectation['threshold'], dtype=np.float32),
        preprocessing=np.asarray(expectation['preprocessing']),
        state_key=np.asarray(expectation['state_key']),
        patch_size=np.asarray(expectation['patch_size'], dtype=np.int16),
        force_atlas_dropout=np.asarray(
            expectation['force_atlas_dropout'], dtype=np.bool_
        ),
        spatial_shape=np.asarray(expectation['spatial_shape'], dtype=np.int32),
        pair_probability_mae=np.asarray(
            pair_probability_mae, dtype=np.float32
        ),
        pair_probability_maximum_error=np.asarray(
            pair_probability_maximum_error, dtype=np.float32
        ),
    )


def infer_one(study_model, spec, row, image, region_map, graph_inputs):
    return audit_sliding_window_probability(
        study_model=study_model,
        row=row,
        image=preprocess_for_spec(image, spec['preprocessing']),
        region_map=region_map,
        graph_inputs=graph_inputs,
        patch_size=spec['patch_size'],
        amp_dtype=spec['amp_dtype'],
        force_atlas_dropout=spec['force_atlas_dropout'],
    )


validation_rows = (
    manifest_df[manifest_df['dev_split'] == 'val']
    .sort_values('subject_id')
    .to_dict('records')
)
if len(validation_rows) != EXPECTED_SPLITS['val']:
    raise RuntimeError('Expected exactly 23 validation subjects.')

loaded_models = {}
checkpoint_metadata = {}
for spec in MODEL_SPECS:
    label = str(spec['model_label'])
    source_label = (
        'v2_best' if label == 'v2_graph_disabled' else label
    )
    if source_label not in loaded_models:
        source_spec = next(
            item for item in MODEL_SPECS
            if item['model_label'] == source_label
        )
        loaded_models[source_label], checkpoint_metadata[source_label] = (
            load_read_only_checkpoint(source_spec)
        )
    checkpoint_metadata[label] = {
        **checkpoint_metadata[source_label],
        **spec,
        'checkpoint_path': checkpoint_metadata[source_label]['checkpoint_path'],
        'checkpoint_sha256': checkpoint_metadata[source_label][
            'checkpoint_sha256'
        ],
    }

print('\nRead-only checkpoint metadata:')
display(pd.DataFrame([
    checkpoint_metadata[label]
    for label in ['v2_best', 'v3_ema_best', 'v4_ema_best']
])[[
    'model_label', 'training_version', 'state_key', 'best_epoch',
    'threshold', 'reported_best_mean_dice', 'patch_size',
    'preprocessing', 'checkpoint_sha256',
]])

case_rows = []
cache_hits = 0
cache_misses = 0
target_arrays_read_splits = set()

for case_index, row in enumerate(validation_rows, start=1):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: validation NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        target_arrays_read_splits.add('val')
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
    spatial_shape = tuple(int(value) for value in target.shape)
    if image.shape != (15, *spatial_shape):
        raise RuntimeError(f'{subject_id}: image/target shape mismatch.')
    region_map, _ = load_region_map(row, spatial_shape)
    graph_inputs = load_graph_inputs(row)

    predictions = {}
    pair_statistics = {}
    v2_spec = MODEL_SPECS[0]
    dropped_spec = MODEL_SPECS[1]
    v2_meta = checkpoint_metadata['v2_best']
    dropped_meta = checkpoint_metadata['v2_graph_disabled']
    v2_path = prediction_cache_path('v2_best', subject_id)
    dropped_path = prediction_cache_path('v2_graph_disabled', subject_id)
    v2_expected = cache_expectation(
        v2_spec, v2_meta, row, spatial_shape
    )
    dropped_expected = cache_expectation(
        dropped_spec, dropped_meta, row, spatial_shape
    )
    v2_cached = load_cached_prediction(v2_path, v2_expected)
    dropped_cached = load_cached_prediction(dropped_path, dropped_expected)
    if v2_cached is not None and dropped_cached is not None:
        predictions['v2_best'] = v2_cached[0]
        predictions['v2_graph_disabled'] = dropped_cached[0]
        pair_statistics['probability_mae'] = v2_cached[1]
        pair_statistics['probability_maximum_error'] = v2_cached[2]
        cache_hits += 2
    else:
        v2_model = loaded_models['v2_best']
        v2_probability = infer_one(
            v2_model, v2_spec, row, image, region_map, graph_inputs
        )
        dropped_probability = infer_one(
            v2_model, dropped_spec, row, image, region_map, graph_inputs
        )
        probability_difference = np.abs(
            v2_probability - dropped_probability
        )
        pair_statistics['probability_mae'] = float(
            probability_difference.mean()
        )
        pair_statistics['probability_maximum_error'] = float(
            probability_difference.max()
        )
        predictions['v2_best'] = (
            v2_probability >= float(v2_meta['threshold'])
        )
        predictions['v2_graph_disabled'] = (
            dropped_probability >= float(dropped_meta['threshold'])
        )
        save_prediction_cache(
            v2_path,
            predictions['v2_best'],
            v2_expected,
            pair_statistics['probability_mae'],
            pair_statistics['probability_maximum_error'],
        )
        save_prediction_cache(
            dropped_path,
            predictions['v2_graph_disabled'],
            dropped_expected,
            pair_statistics['probability_mae'],
            pair_statistics['probability_maximum_error'],
        )
        cache_misses += 2
        del v2_probability, dropped_probability, probability_difference

    for spec in MODEL_SPECS[2:]:
        label = str(spec['model_label'])
        metadata = checkpoint_metadata[label]
        cache_path = prediction_cache_path(label, subject_id)
        expectation = cache_expectation(
            spec, metadata, row, spatial_shape
        )
        cached = load_cached_prediction(cache_path, expectation)
        if cached is not None:
            predictions[label] = cached[0]
            cache_hits += 1
        else:
            probability = infer_one(
                loaded_models[label],
                spec,
                row,
                image,
                region_map,
                graph_inputs,
            )
            predictions[label] = (
                probability >= float(metadata['threshold'])
            )
            save_prediction_cache(
                cache_path, predictions[label], expectation
            )
            cache_misses += 1
            del probability

    v2_prediction = predictions['v2_best']
    dropped_prediction = predictions['v2_graph_disabled']
    prediction_union = int((v2_prediction | dropped_prediction).sum())
    changed_voxels = int((v2_prediction ^ dropped_prediction).sum())
    pair_statistics['changed_voxels'] = changed_voxels
    pair_statistics['changed_fraction_of_union'] = (
        changed_voxels / max(prediction_union, 1)
    )
    prediction_denominator = int(v2_prediction.sum() + dropped_prediction.sum())
    pair_statistics['prediction_dice'] = (
        1.0
        if prediction_denominator == 0
        else 2.0 * int((v2_prediction & dropped_prediction).sum())
        / prediction_denominator
    )

    lesion_bin = str(row.get('lesion_volume_bin', 'unknown'))
    atlas_mode = str(row.get('trial8_atlas_mode', 'unknown'))
    center_value = 'unavailable'
    for center_column in [
        'center', 'center_id', 'site', 'site_id', 'institution',
    ]:
        if center_column in row and pd.notna(row[center_column]):
            center_value = str(row[center_column])
            break
    for spec in MODEL_SPECS:
        label = str(spec['model_label'])
        metrics = official_case_metrics(
            predictions[label], target, spacing
        )
        case_rows.append({
            'subject_id': subject_id,
            'dev_split': 'val',
            'model_label': label,
            'training_version': checkpoint_metadata[label][
                'training_version'
            ],
            'checkpoint_sha256': checkpoint_metadata[label][
                'checkpoint_sha256'
            ],
            'state_key': spec['state_key'],
            'threshold': float(checkpoint_metadata[label]['threshold']),
            'preprocessing': spec['preprocessing'],
            'patch_size': 'x'.join(str(value) for value in spec['patch_size']),
            'force_atlas_dropout': bool(spec['force_atlas_dropout']),
            'lesion_volume_bin': lesion_bin,
            'under_5_ml': bool(metrics['target_volume_ml'] < 5.0),
            'empty_target': bool(metrics['target_voxels'] == 0),
            'atlas_mode': atlas_mode,
            'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
            'center': center_value,
            'graph_pair_probability_mae': (
                pair_statistics['probability_mae']
                if label in {'v2_best', 'v2_graph_disabled'}
                else np.nan
            ),
            'graph_pair_probability_maximum_error': (
                pair_statistics['probability_maximum_error']
                if label in {'v2_best', 'v2_graph_disabled'}
                else np.nan
            ),
            'graph_pair_changed_voxels': (
                pair_statistics['changed_voxels']
                if label in {'v2_best', 'v2_graph_disabled'}
                else np.nan
            ),
            'graph_pair_changed_fraction_of_union': (
                pair_statistics['changed_fraction_of_union']
                if label in {'v2_best', 'v2_graph_disabled'}
                else np.nan
            ),
            'graph_pair_prediction_dice': (
                pair_statistics['prediction_dice']
                if label in {'v2_best', 'v2_graph_disabled'}
                else np.nan
            ),
            'target_arrays_read_from_split': 'val',
            **metrics,
        })

    del image, target, region_map, graph_inputs, predictions
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if case_index == 1 or case_index % 5 == 0 or case_index == len(validation_rows):
        print(
            f'Audited {case_index}/{len(validation_rows)}: {subject_id}'
        )

case_metrics_df = pd.DataFrame(case_rows)


def summarize_models(frame):
    summary = (
        frame.groupby('model_label', as_index=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_dice=('dice', 'mean'),
            standard_deviation_dice=('dice', 'std'),
            median_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_difference_ml=(
                'absolute_volume_difference_ml', 'mean'
            ),
            mean_lesion_f1=('lesion_f1', 'mean'),
            mean_absolute_lesion_count_difference=(
                'absolute_lesion_count_difference', 'mean'
            ),
            total_intersection=('intersection', 'sum'),
            total_predicted_voxels=('predicted_voxels', 'sum'),
            total_target_voxels=('target_voxels', 'sum'),
        )
    )
    summary['global_dice_diagnostic'] = (
        2.0 * summary['total_intersection']
        / np.maximum(
            summary['total_predicted_voxels']
            + summary['total_target_voxels'],
            1,
        )
    )
    return summary.sort_values('mean_dice', ascending=False).reset_index(drop=True)


def grouped_summary(frame, grouping):
    return (
        frame.groupby(['model_label', grouping], dropna=False, as_index=False)
        .agg(
            subjects=('subject_id', 'count'),
            mean_dice=('dice', 'mean'),
            median_dice=('dice', 'median'),
            mean_precision=('precision', 'mean'),
            mean_recall=('recall', 'mean'),
            mean_absolute_volume_difference_ml=(
                'absolute_volume_difference_ml', 'mean'
            ),
            mean_lesion_f1=('lesion_f1', 'mean'),
            mean_absolute_lesion_count_difference=(
                'absolute_lesion_count_difference', 'mean'
            ),
        )
        .sort_values(['model_label', grouping])
        .reset_index(drop=True)
    )


model_summary_df = summarize_models(case_metrics_df)
lesion_bin_summary_df = grouped_summary(
    case_metrics_df, 'lesion_volume_bin'
)
atlas_mode_summary_df = grouped_summary(case_metrics_df, 'atlas_mode')

rank_rows = []
for subject_id, subject_frame in case_metrics_df.groupby('subject_id'):
    subject_frame = subject_frame.copy()
    subject_frame['dice_rank'] = rankdata(
        -subject_frame['dice'].to_numpy(), method='average'
    )
    subject_frame['avd_rank'] = rankdata(
        subject_frame['absolute_volume_difference_ml'].to_numpy(),
        method='average',
    )
    subject_frame['lesion_f1_rank'] = rankdata(
        -subject_frame['lesion_f1'].to_numpy(), method='average'
    )
    subject_frame['alcd_rank'] = rankdata(
        subject_frame['absolute_lesion_count_difference'].to_numpy(),
        method='average',
    )
    subject_frame['per_case_mean_rank'] = subject_frame[
        ['dice_rank', 'avd_rank', 'lesion_f1_rank', 'alcd_rank']
    ].mean(axis=1)
    rank_rows.extend(subject_frame[[
        'subject_id', 'model_label', 'dice_rank', 'avd_rank',
        'lesion_f1_rank', 'alcd_rank', 'per_case_mean_rank',
    ]].to_dict('records'))
rank_case_df = pd.DataFrame(rank_rows)
local_rank_df = (
    rank_case_df.groupby('model_label', as_index=False)
    .agg(
        validation_subjects=('subject_id', 'count'),
        mean_local_rank=('per_case_mean_rank', 'mean'),
        mean_dice_rank=('dice_rank', 'mean'),
        mean_avd_rank=('avd_rank', 'mean'),
        mean_lesion_f1_rank=('lesion_f1_rank', 'mean'),
        mean_alcd_rank=('alcd_rank', 'mean'),
    )
    .sort_values('mean_local_rank')
    .reset_index(drop=True)
)
local_rank_df.insert(0, 'local_final_rank', np.arange(1, len(local_rank_df) + 1))

v2_normal = case_metrics_df[
    case_metrics_df['model_label'] == 'v2_best'
].copy()
v2_dropped = case_metrics_df[
    case_metrics_df['model_label'] == 'v2_graph_disabled'
].copy()
graph_counterfactual_df = v2_normal.merge(
    v2_dropped,
    on='subject_id',
    suffixes=('_normal', '_graph_disabled'),
    validate='one_to_one',
)
graph_counterfactual_df['dice_delta_normal_minus_disabled'] = (
    graph_counterfactual_df['dice_normal']
    - graph_counterfactual_df['dice_graph_disabled']
)
graph_counterfactual_df['lesion_f1_delta_normal_minus_disabled'] = (
    graph_counterfactual_df['lesion_f1_normal']
    - graph_counterfactual_df['lesion_f1_graph_disabled']
)
graph_counterfactual_df['avd_improvement_normal_minus_disabled'] = (
    graph_counterfactual_df['absolute_volume_difference_ml_graph_disabled']
    - graph_counterfactual_df['absolute_volume_difference_ml_normal']
)

worst_cases_df = (
    case_metrics_df.sort_values(['model_label', 'dice', 'lesion_f1'])
    .groupby('model_label', as_index=False, group_keys=False)
    .head(8)
    .reset_index(drop=True)
)

metric_columns = [
    'dice', 'precision', 'recall', 'absolute_volume_difference_ml',
    'lesion_f1', 'absolute_lesion_count_difference',
]
source_checkpoint_hashes_unchanged = all(
    file_sha256(Path(metadata['checkpoint_path']))
    == metadata['checkpoint_sha256']
    for label, metadata in checkpoint_metadata.items()
    if label != 'v2_graph_disabled'
)
audit_checks = {
    'exactly_23_validation_subjects': (
        case_metrics_df['subject_id'].nunique() == 23
    ),
    'four_configurations_complete': (
        case_metrics_df['model_label'].nunique() == 4
        and len(case_metrics_df) == 23 * 4
    ),
    'only_validation_targets_read': set(
        case_metrics_df['target_arrays_read_from_split'].astype(str)
    ) == {'val'},
    'test_targets_not_read': 'test' not in target_arrays_read_splits,
    'all_metrics_finite': bool(np.isfinite(
        case_metrics_df[metric_columns].to_numpy(dtype=np.float64)
    ).all()),
    'probabilistic_metrics_bounded': bool(
        case_metrics_df[['dice', 'precision', 'recall', 'lesion_f1']]
        .ge(0.0).all().all()
        and case_metrics_df[['dice', 'precision', 'recall', 'lesion_f1']]
        .le(1.0).all().all()
    ),
    'official_iou_threshold_is_0p20': np.isclose(
        IOU_MATCH_THRESHOLD, 0.20
    ),
    'component_connectivity_is_26': COMPONENT_CONNECTIVITY == 26,
    'v2_graph_counterfactual_complete': (
        len(graph_counterfactual_df) == 23
    ),
    'source_checkpoints_unchanged': source_checkpoint_hashes_unchanged,
    'no_duplicate_case_configuration_rows': not case_metrics_df.duplicated(
        ['subject_id', 'model_label']
    ).any(),
    'no_training_performed': True,
    'model_training_not_authorized_by_this_block': True,
}
audit_passed = bool(all(audit_checks.values()))

atomic_write_csv(case_metrics_df, CASE_METRICS_PATH)
atomic_write_csv(model_summary_df, MODEL_SUMMARY_PATH)
atomic_write_csv(lesion_bin_summary_df, LESION_BIN_SUMMARY_PATH)
atomic_write_csv(atlas_mode_summary_df, ATLAS_MODE_SUMMARY_PATH)
atomic_write_csv(local_rank_df, LOCAL_RANK_PATH)
atomic_write_csv(graph_counterfactual_df, GRAPH_COUNTERFACTUAL_PATH)
atomic_write_csv(worst_cases_df, WORST_CASES_PATH)

graph_summary = {
    'mean_probability_mae': float(
        graph_counterfactual_df['graph_pair_probability_mae_normal'].mean()
    ),
    'maximum_probability_error': float(
        graph_counterfactual_df[
            'graph_pair_probability_maximum_error_normal'
        ].max()
    ),
    'mean_prediction_dice': float(
        graph_counterfactual_df['graph_pair_prediction_dice_normal'].mean()
    ),
    'subjects_with_changed_binary_prediction': int((
        graph_counterfactual_df['graph_pair_changed_voxels_normal'] > 0
    ).sum()),
    'mean_dice_delta_normal_minus_disabled': float(
        graph_counterfactual_df[
            'dice_delta_normal_minus_disabled'
        ].mean()
    ),
    'mean_lesion_f1_delta_normal_minus_disabled': float(
        graph_counterfactual_df[
            'lesion_f1_delta_normal_minus_disabled'
        ].mean()
    ),
}

atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'audit_version': AUDIT_VERSION,
        'audit_passed': audit_passed,
        'checks': audit_checks,
        'validation_subjects': 23,
        'test_targets_read': False,
        'training_performed': False,
        'model_training_authorized': False,
        'study_trainable_parameters': 0,
        'official_metric_policy': {
            'dice': 'voxel Dice; empty-empty equals 1',
            'absolute_volume_difference': 'milliliters',
            'lesion_instances': '26-connected components',
            'lesion_match': 'maximum bipartite matching at IoU >= 0.20',
            'lesion_f1': '2TP/(2TP+FP+FN); empty-empty equals 1',
            'absolute_lesion_count_difference': 'absolute component-count difference',
        },
        'checkpoint_metadata': [
            to_builtin(checkpoint_metadata[label])
            for label in [
                'v2_best', 'v2_graph_disabled',
                'v3_ema_best', 'v4_ema_best',
            ]
        ],
        'prediction_cache_hits': cache_hits,
        'prediction_cache_misses': cache_misses,
        'graph_counterfactual_summary': graph_summary,
        'next_block_authorized': audit_passed,
        'next_block': 'target-independent crop and development-fold audit',
    },
    AUDIT_GATE_PATH,
)

print('\nOfficial-metric validation summary:')
display(model_summary_df[[
    'model_label', 'subjects', 'mean_dice', 'standard_deviation_dice',
    'median_dice', 'global_dice_diagnostic', 'mean_precision',
    'mean_recall', 'mean_absolute_volume_difference_ml',
    'mean_lesion_f1', 'mean_absolute_lesion_count_difference',
]])

print('\nLocal rank-then-aggregate comparison:')
display(local_rank_df)

print('\nPerformance by lesion-volume bin:')
display(lesion_bin_summary_df)

print('\nPerformance by atlas mode:')
display(atlas_mode_summary_df)

print('\nV2 graph counterfactual summary:')
display(pd.DataFrame([
    {'metric': name, 'value': value}
    for name, value in graph_summary.items()
]))

print('\nWorst validation cases by configuration:')
display(worst_cases_df[[
    'model_label', 'subject_id', 'lesion_volume_bin', 'target_volume_ml',
    'dice', 'precision', 'recall', 'lesion_f1',
    'absolute_volume_difference_ml',
    'absolute_lesion_count_difference', 'atlas_mode',
]])

print('\nAudit checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in audit_checks.items()
]))

if not audit_passed:
    failed = [name for name, passed in audit_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 OFFICIAL-METRIC FAILURE AUDIT FAILED. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 OFFICIAL-METRIC FAILURE AUDIT PASSED')
print('=' * 80)
print('Validation subjects:                23')
print('Test targets read:                  False')
print('Training performed:                 False')
print('Model training authorized:          False')
print('Study trainable parameters:         0 (diagnostic only)')
print(f'Prediction cache hits:              {cache_hits}')
print(f'Prediction cache misses:            {cache_misses}')
print(f'Case metrics:                       {CASE_METRICS_PATH}')
print(f'Model summary:                      {MODEL_SUMMARY_PATH}')
print(f'Local rank summary:                 {LOCAL_RANK_PATH}')
print(f'Graph counterfactual:               {GRAPH_COUNTERFACTUAL_PATH}')
print(f'Audit gate:                         {AUDIT_GATE_PATH}')
print('\nNo checkpoint or frozen data artifact was modified.')
print('Next: check this output before the crop and fold-integrity block.')


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
ensure_package('scipy')

from scipy import ndimage
from scipy.stats import spearmanr
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
GRAPH_FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BLOCK1_VERSION = 'v1_official_metrics_v2_v3_v4_graph_counterfactual'
AUDIT_VERSION = 'v1_crop_boundary_and_train5fold_integrity'

EXPECTED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
NUMBER_OF_FOLDS = 5
FOLD_SEED = 20260924
NEAR_CROP_BOUNDARY_MM = 4.0
MINIMUM_ESTIMATED_BBOX_COVERAGE = 0.995

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_VERIFICATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / GRAPH_FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
)
GRAPH_VERIFICATION_GATE_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_verification_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'official_metric_audit_gate.json'
BLOCK1_CASE_METRICS_PATH = BLOCK1_ROOT / 'validation_case_metrics.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / AUDIT_VERSION
)
CROP_AUDIT_PATH = OUTPUT_ROOT / 'crop_boundary_integrity_audit.csv'
CROP_RISK_PATH = OUTPUT_ROOT / 'crop_risk_cases.csv'
DUPLICATE_AUDIT_PATH = OUTPUT_ROOT / 'exact_input_duplicate_audit.csv'
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest.csv'
)
FOLD_BALANCE_PATH = OUTPUT_ROOT / 'fold_balance_summary.csv'
FOLD_STRATA_PATH = OUTPUT_ROOT / 'fold_strata_counts.csv'
VALIDATION_ASSOCIATION_PATH = (
    OUTPUT_ROOT / 'validation_failure_associations.csv'
)
AUDIT_GATE_PATH = OUTPUT_ROOT / 'crop_fold_integrity_gate.json'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def array_sha256(*arrays):
    digest = hashlib.sha256()
    for array in arrays:
        contiguous = np.ascontiguousarray(array)
        digest.update(str(contiguous.shape).encode('utf-8'))
        digest.update(str(contiguous.dtype).encode('utf-8'))
        digest.update(contiguous.view(np.uint8))
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def scalar_float(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).item())


def scalar_int(data, name, default=0):
    if name not in data.files:
        return int(default)
    return int(np.asarray(data[name]).item())


def lesion_volume_bin(volume_ml):
    if volume_ml <= 0.0:
        return 'empty'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    if volume_ml < 75.0:
        return 'large'
    return 'very_large'


def safe_spearman(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    valid = np.isfinite(x) & np.isfinite(y)
    if int(valid.sum()) < 3:
        return np.nan, np.nan, int(valid.sum())
    if np.unique(x[valid]).size < 2 or np.unique(y[valid]).size < 2:
        return np.nan, np.nan, int(valid.sum())
    result = spearmanr(x[valid], y[valid])
    return float(result.statistic), float(result.pvalue), int(valid.sum())




atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'audit_version': AUDIT_VERSION,
        'audit_passed': False,
        'status': 'running_or_incomplete',
        'test_target_arrays_read': False,
        'training_performed': False,
        'model_training_authorized': False,
    },
    AUDIT_GATE_PATH,
)






required_paths = [
    VERIFIED_MANIFEST_PATH,
    GRAPH_VERIFICATION_GATE_PATH,
    BLOCK1_GATE_PATH,
    BLOCK1_CASE_METRICS_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 Block 2 prerequisites are missing:\n' + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
if verified_manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified model manifest changed. '
        f'Observed {verified_manifest_sha256}; expected {EXPECTED_MANIFEST_SHA256}.'
    )

graph_gate = json.loads(
    GRAPH_VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
block1_gate = json.loads(BLOCK1_GATE_PATH.read_text(encoding='utf-8'))
if graph_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if graph_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification was not target-free.')
if block1_gate.get('audit_passed') is not True:
    raise RuntimeError('Block 1 official-metric audit is not passed.')
if block1_gate.get('test_targets_read') is not False:
    raise RuntimeError('Block 1 reports test-target access.')
if block1_gate.get('training_performed') is not False:
    raise RuntimeError('Block 1 unexpectedly reports training.')



manifest_header = pd.read_csv(VERIFIED_MANIFEST_PATH, nrows=0).columns.tolist()
base_columns = [
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
]
missing_columns = sorted(set(base_columns) - set(manifest_header))
if missing_columns:
    raise RuntimeError(f'Verified manifest is missing columns: {missing_columns}')

center_candidates = [
    'center', 'center_id', 'site', 'site_id', 'institution', 'hospital',
    'acquisition_center', 'scanner_site', 'manufacturer',
]
center_columns = [name for name in center_candidates if name in manifest_header]
safe_manifest_columns = base_columns + center_columns
manifest_df = pd.read_csv(
    VERIFIED_MANIFEST_PATH,
    usecols=safe_manifest_columns,
).sort_values('subject_id').reset_index(drop=True)

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_are_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'all_npz_paths_exist': manifest_df['vascufate_v2_npz_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Frozen manifest checks failed: ' + ', '.join(failed))

print('Trial8 Block 2 upstream gates passed.')
print(f'Frozen subjects:                    {len(manifest_df)}')
print(f'Split counts:                       {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Frozen manifest SHA-256:            {verified_manifest_sha256}')
print(f'Available target-free center fields:{center_columns or ["none"]}')
print('Test target arrays authorized:      False')
print('Study trainable parameters:         0 (audit only)')






connectivity_26 = np.ones((3, 3, 3), dtype=np.uint8)
case_rows = []
target_arrays_read_splits = set()

for case_index, row in enumerate(manifest_df.to_dict('records'), start=1):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    observed_npz_sha256 = file_sha256(npz_path)
    if observed_npz_sha256 != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')

    with np.load(npz_path, allow_pickle=False) as data:
        target_free_required = {
            'image', 'crop_mask', 'crop_origin', 'full_resampled_shape',
            'target_spacing', 'affine', 'full_resampled_affine',
        }
        missing = sorted(target_free_required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing target-free arrays: {missing}')

        image = np.asarray(data['image'])
        crop_mask = np.asarray(data['crop_mask']) > 0
        crop_origin = np.asarray(data['crop_origin'], dtype=np.int64)
        full_shape = np.asarray(data['full_resampled_shape'], dtype=np.int64)
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        affine = np.asarray(data['affine'], dtype=np.float64)
        full_affine = np.asarray(data['full_resampled_affine'], dtype=np.float64)

        if image.ndim != 4 or image.shape[0] != 15:
            raise RuntimeError(f'{subject_id}: expected a 15-channel 3D image.')
        spatial_shape = np.asarray(image.shape[1:], dtype=np.int64)
        if crop_mask.shape != tuple(spatial_shape):
            raise RuntimeError(f'{subject_id}: crop-mask shape mismatch.')
        if crop_origin.shape != (3,) or full_shape.shape != (3,):
            raise RuntimeError(f'{subject_id}: crop geometry is malformed.')
        if spacing.shape != (3,) or np.any(spacing <= 0):
            raise RuntimeError(f'{subject_id}: target spacing is malformed.')
        if not np.isfinite(image).all():
            raise RuntimeError(f'{subject_id}: image contains non-finite values.')
        if not np.isfinite(affine).all() or not np.isfinite(full_affine).all():
            raise RuntimeError(f'{subject_id}: affine contains non-finite values.')

        crop_stop = crop_origin + spatial_shape
        geometry_inside_full_grid = bool(
            np.all(crop_origin >= 0) and np.all(crop_stop <= full_shape)
        )
        expected_translation = full_affine[:3, :3] @ crop_origin + full_affine[:3, 3]
        affine_translation_error = float(
            np.max(np.abs(affine[:3, 3] - expected_translation))
        )
        input_fingerprint = array_sha256(
            image[[0, 1, 2]], affine, spacing.astype(np.float32)
        )

        target_read = split in {'train', 'val'}
        target_voxels = np.nan
        target_volume_ml = np.nan
        target_bin = ''
        target_components = np.nan
        largest_component_fraction = np.nan
        target_touches_crop_boundary = False
        target_near_crop_boundary = False
        minimum_target_edge_clearance_mm = np.nan
        cropped_target_support_fraction = np.nan
        stored_target_support_coverage = np.nan
        estimated_bbox_target_coverage = np.nan
        source_target_volume_ml = np.nan
        stored_target_binary_volume_ml = np.nan
        binary_loss_weight = np.nan
        source_nonempty_became_empty = False
        target_volume_metadata_error_ml = np.nan

        if target_read:
            if 'target_binary' not in data.files:
                raise RuntimeError(f'{subject_id}: target_binary is missing.')
            target_arrays_read_splits.add(split)
            target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
            if target.shape != tuple(spatial_shape):
                raise RuntimeError(f'{subject_id}: target/image shape mismatch.')

            target_voxels = int(target.sum())
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            target_volume_ml = float(target_voxels * voxel_volume_ml)
            target_bin = lesion_volume_bin(target_volume_ml)
            stored_target_support_coverage = scalar_float(
                data, 'target_crop_coverage'
            )
            source_target_volume_ml = scalar_float(
                data, 'source_target_volume_ml'
            )
            stored_target_binary_volume_ml = scalar_float(
                data, 'target_binary_volume_ml'
            )
            binary_loss_weight = scalar_float(data, 'binary_loss_weight', 1.0)
            target_volume_metadata_error_ml = abs(
                stored_target_binary_volume_ml - target_volume_ml
            )
            source_nonempty_became_empty = bool(
                source_target_volume_ml > 0.0 and target_voxels == 0
            )

            if target_voxels > 0:
                overlap_with_support = int((target & crop_mask).sum())
                cropped_target_support_fraction = float(
                    overlap_with_support / target_voxels
                )
                if (
                    np.isfinite(stored_target_support_coverage)
                    and stored_target_support_coverage > 0.0
                ):
                    estimated_full_target_voxels = (
                        overlap_with_support / stored_target_support_coverage
                    )
                    estimated_bbox_target_coverage = float(np.clip(
                        target_voxels / max(estimated_full_target_voxels, 1.0),
                        0.0,
                        1.0,
                    ))

                coordinates = np.argwhere(target)
                minimum = coordinates.min(axis=0)
                maximum = coordinates.max(axis=0)
                lower_clearance_mm = minimum * spacing
                upper_clearance_mm = (spatial_shape - 1 - maximum) * spacing
                minimum_target_edge_clearance_mm = float(np.min(
                    np.concatenate([lower_clearance_mm, upper_clearance_mm])
                ))
                target_touches_crop_boundary = bool(
                    minimum_target_edge_clearance_mm <= 1e-8
                )
                target_near_crop_boundary = bool(
                    minimum_target_edge_clearance_mm < NEAR_CROP_BOUNDARY_MM
                )

                component_labels, component_count = ndimage.label(
                    target, structure=connectivity_26
                )
                component_sizes = np.bincount(
                    component_labels.reshape(-1),
                    minlength=component_count + 1,
                )[1:]
                target_components = int(component_count)
                largest_component_fraction = float(
                    component_sizes.max() / target_voxels
                )
            else:
                target_components = 0
                largest_component_fraction = 1.0

    center_value = 'unavailable'
    if center_columns:
        center_parts = []
        for column in center_columns:
            value = row.get(column, '')
            if pd.notna(value) and str(value).strip():
                center_parts.append(f'{column}={value}')
        if center_parts:
            center_value = '|'.join(center_parts)

    case_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'npz_path': str(npz_path),
        'npz_sha256': observed_npz_sha256,
        'input_fingerprint_ncct_broad_cta': input_fingerprint,
        'atlas_mode': str(row['trial8_atlas_mode']),
        'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
        'center_metadata': center_value,
        'image_shape': str(tuple(int(value) for value in image.shape)),
        'spatial_shape': str(tuple(int(value) for value in spatial_shape)),
        'crop_origin': str(tuple(int(value) for value in crop_origin)),
        'crop_stop': str(tuple(int(value) for value in crop_stop)),
        'full_resampled_shape': str(tuple(int(value) for value in full_shape)),
        'crop_voxels': int(np.prod(spatial_shape)),
        'full_grid_voxels': int(np.prod(full_shape)),
        'crop_reduction_fraction': float(
            1.0 - np.prod(spatial_shape) / np.prod(full_shape)
        ),
        'crop_mask_voxels': int(crop_mask.sum()),
        'geometry_inside_full_grid': geometry_inside_full_grid,
        'affine_translation_error': affine_translation_error,
        'target_array_read': target_read,
        'target_voxels': target_voxels,
        'target_volume_ml': target_volume_ml,
        'lesion_volume_bin': target_bin,
        'target_components_26n': target_components,
        'largest_component_fraction': largest_component_fraction,
        'target_touches_crop_boundary': target_touches_crop_boundary,
        'target_near_crop_boundary_under_4mm': target_near_crop_boundary,
        'minimum_target_edge_clearance_mm': minimum_target_edge_clearance_mm,
        'cropped_target_inside_support_fraction': (
            cropped_target_support_fraction
        ),
        'stored_full_target_support_coverage': (
            stored_target_support_coverage
        ),
        'estimated_bbox_target_coverage': estimated_bbox_target_coverage,
        'source_target_volume_ml': source_target_volume_ml,
        'stored_target_binary_volume_ml': stored_target_binary_volume_ml,
        'target_volume_metadata_error_ml': target_volume_metadata_error_ml,
        'binary_loss_weight': binary_loss_weight,
        'source_nonempty_became_cropped_empty': source_nonempty_became_empty,
    })

    if case_index == 1 or case_index % 20 == 0 or case_index == len(manifest_df):
        print(
            f'Checked {case_index}/{len(manifest_df)}: {subject_id}, '
            f'split={split}, target_read={target_read}'
        )

crop_audit_df = pd.DataFrame(case_rows)
atomic_write_csv(crop_audit_df, CROP_AUDIT_PATH)






duplicate_rows = []
for fingerprint_type, column in [
    ('full_npz_sha256', 'npz_sha256'),
    ('input_ncct_broad_cta_sha256', 'input_fingerprint_ncct_broad_cta'),
]:
    for fingerprint, group in crop_audit_df.groupby(column):
        if len(group) <= 1:
            continue
        duplicate_rows.append({
            'fingerprint_type': fingerprint_type,
            'fingerprint': fingerprint,
            'subjects': '|'.join(sorted(group['subject_id'].astype(str))),
            'splits': '|'.join(sorted(group['dev_split'].astype(str).unique())),
            'subject_count': int(len(group)),
            'cross_split': bool(group['dev_split'].nunique() > 1),
        })

duplicate_audit_df = pd.DataFrame(duplicate_rows, columns=[
    'fingerprint_type', 'fingerprint', 'subjects', 'splits',
    'subject_count', 'cross_split',
])
atomic_write_csv(duplicate_audit_df, DUPLICATE_AUDIT_PATH)






train_audit_df = (
    crop_audit_df[crop_audit_df['dev_split'] == 'train']
    .copy()
    .sort_values('subject_id')
    .reset_index(drop=True)
)
if len(train_audit_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Expected exactly 102 train subjects for fold creation.')
if not train_audit_df['target_array_read'].all():
    raise RuntimeError('Train targets were not completely audited.')

bin_frequency = Counter(train_audit_df['lesion_volume_bin'].astype(str))
mode_frequency = Counter(train_audit_df['atlas_mode'].astype(str))
center_frequency = Counter(train_audit_df['center_metadata'].astype(str))




assignment_records = train_audit_df.to_dict('records')
assignment_records.sort(key=lambda record: (
    mode_frequency[str(record['atlas_mode'])],
    bin_frequency[str(record['lesion_volume_bin'])],
    center_frequency[str(record['center_metadata'])],
    -float(record['target_volume_ml']),
    str(record['subject_id']),
))

fold_size = Counter()
fold_bin_count = defaultdict(Counter)
fold_mode_count = defaultdict(Counter)
fold_center_count = defaultdict(Counter)
fold_log_volume = defaultdict(float)
fold_assignments = {}

for record in assignment_records:
    subject_id = str(record['subject_id'])
    lesion_bin = str(record['lesion_volume_bin'])
    atlas_mode = str(record['atlas_mode'])
    center = str(record['center_metadata'])
    log_volume = math.log1p(float(record['target_volume_ml']))
    rotation = int(hashlib.sha256(
        f'{FOLD_SEED}:{subject_id}'.encode('utf-8')
    ).hexdigest()[:8], 16) % NUMBER_OF_FOLDS
    fold_order = [
        (rotation + offset) % NUMBER_OF_FOLDS
        for offset in range(NUMBER_OF_FOLDS)
    ]
    selected_fold = min(
        fold_order,
        key=lambda fold: (
            fold_bin_count[fold][lesion_bin],
            fold_mode_count[fold][atlas_mode],
            fold_center_count[fold][center],
            fold_size[fold],
            fold_log_volume[fold],
            fold_order.index(fold),
        ),
    )
    fold_assignments[subject_id] = int(selected_fold)
    fold_size[selected_fold] += 1
    fold_bin_count[selected_fold][lesion_bin] += 1
    fold_mode_count[selected_fold][atlas_mode] += 1
    fold_center_count[selected_fold][center] += 1
    fold_log_volume[selected_fold] += log_volume

fold_rows = []
train_lookup = train_audit_df.set_index('subject_id').to_dict('index')
for row in manifest_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    if split == 'train':
        audit = train_lookup[subject_id]
        development_fold = int(fold_assignments[subject_id])
        development_role = 'cross_validation_pool'
        train_only_bin = str(audit['lesion_volume_bin'])
        train_only_volume = float(audit['target_volume_ml'])
        train_only_components = int(audit['target_components_26n'])
    elif split == 'val':
        development_fold = -1
        development_role = 'locked_validation_confirmation'
        train_only_bin = ''
        train_only_volume = np.nan
        train_only_components = -1
    else:
        development_fold = -1
        development_role = 'sealed_test'
        train_only_bin = ''
        train_only_volume = np.nan
        train_only_components = -1
    fold_rows.append({
        'subject_id': subject_id,
        'frozen_dev_split': split,
        'development_role': development_role,
        'development_fold': development_fold,
        'train_only_lesion_volume_bin': train_only_bin,
        'train_only_target_volume_ml': train_only_volume,
        'train_only_target_components_26n': train_only_components,
        'atlas_mode': str(row['trial8_atlas_mode']),
        'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
        'center_metadata': (
            train_lookup[subject_id]['center_metadata']
            if split == 'train'
            else crop_audit_df.set_index('subject_id').loc[
                subject_id, 'center_metadata'
            ]
        ),
        'source_verified_manifest_sha256': verified_manifest_sha256,
        'fold_seed': FOLD_SEED,
        'fold_policy': 'train_only_balanced_greedy_v1',
    })

fold_manifest_df = pd.DataFrame(fold_rows).sort_values('subject_id')

train_fold_df = fold_manifest_df[
    fold_manifest_df['frozen_dev_split'] == 'train'
].copy()
fold_balance_df = (
    train_fold_df.groupby('development_fold', as_index=False)
    .agg(
        subjects=('subject_id', 'count'),
        target_volume_mean_ml=('train_only_target_volume_ml', 'mean'),
        target_volume_median_ml=('train_only_target_volume_ml', 'median'),
        target_volume_sum_ml=('train_only_target_volume_ml', 'sum'),
        mean_lesion_components=(
            'train_only_target_components_26n', 'mean'
        ),
        atlas_enabled_subjects=('atlas_enabled', 'sum'),
    )
    .sort_values('development_fold')
    .reset_index(drop=True)
)

strata_rows = []
for stratum_name, column in [
    ('lesion_volume_bin', 'train_only_lesion_volume_bin'),
    ('atlas_mode', 'atlas_mode'),
    ('center_metadata', 'center_metadata'),
]:
    counts = (
        train_fold_df.groupby(['development_fold', column])
        .size()
        .unstack(fill_value=0)
        .reindex(range(NUMBER_OF_FOLDS), fill_value=0)
    )
    for category in counts.columns:
        for fold in range(NUMBER_OF_FOLDS):
            strata_rows.append({
                'stratum': stratum_name,
                'category': str(category),
                'development_fold': fold,
                'subjects': int(counts.loc[fold, category]),
                'category_total': int(counts[category].sum()),
                'maximum_minus_minimum': int(
                    counts[category].max() - counts[category].min()
                ),
            })
fold_strata_df = pd.DataFrame(strata_rows)


def stratum_balanced(stratum, minimum_total=1):
    subset = fold_strata_df[
        (fold_strata_df['stratum'] == stratum)
        & (fold_strata_df['category_total'] >= minimum_total)
    ]
    return bool(subset['maximum_minus_minimum'].le(1).all())


fold_checks = {
    'exactly_102_train_subjects_assigned': len(train_fold_df) == 102,
    'every_train_subject_assigned_once': (
        train_fold_df['subject_id'].nunique() == 102
        and train_fold_df['development_fold'].between(0, 4).all()
    ),
    'validation_and_test_have_no_development_fold': fold_manifest_df.loc[
        fold_manifest_df['frozen_dev_split'].isin(['val', 'test']),
        'development_fold',
    ].eq(-1).all(),
    'fold_sizes_differ_by_at_most_one': (
        fold_balance_df['subjects'].max()
        - fold_balance_df['subjects'].min()
    ) <= 1,
    'lesion_volume_bins_differ_by_at_most_one': stratum_balanced(
        'lesion_volume_bin'
    ),
    'atlas_modes_with_five_or_more_cases_are_balanced': stratum_balanced(
        'atlas_mode', minimum_total=5
    ),
    'center_categories_with_five_or_more_cases_are_balanced': stratum_balanced(
        'center_metadata', minimum_total=5
    ),
    'every_fold_contains_nonempty_lesions': bool(
        train_fold_df.groupby('development_fold')['train_only_target_volume_ml']
        .apply(lambda values: (values > 0).any()).all()
    ),
}

atomic_write_csv(fold_manifest_df, FOLD_MANIFEST_PATH)
atomic_write_csv(fold_balance_df, FOLD_BALANCE_PATH)
atomic_write_csv(fold_strata_df, FOLD_STRATA_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)






block1_metrics_df = pd.read_csv(BLOCK1_CASE_METRICS_PATH)
v2_metrics_df = block1_metrics_df[
    block1_metrics_df['model_label'] == 'v2_best'
].copy()
validation_crop_df = crop_audit_df[
    crop_audit_df['dev_split'] == 'val'
].copy()
validation_join_df = v2_metrics_df.merge(
    validation_crop_df,
    on='subject_id',
    validate='one_to_one',
    suffixes=('_metric', '_crop'),
)

association_rows = []
for predictor in [
    'minimum_target_edge_clearance_mm',
    'cropped_target_inside_support_fraction',
    'estimated_bbox_target_coverage',
    'target_volume_ml_crop',
    'target_components_26n',
]:
    correlation, p_value, subjects = safe_spearman(
        validation_join_df[predictor], validation_join_df['dice']
    )
    association_rows.append({
        'outcome': 'v2_validation_dice',
        'predictor': predictor,
        'subjects': subjects,
        'spearman_rho': correlation,
        'two_sided_p_value': p_value,
        'exploratory_only': True,
    })
validation_association_df = pd.DataFrame(association_rows)
atomic_write_csv(validation_association_df, VALIDATION_ASSOCIATION_PATH)






target_audited_df = crop_audit_df[
    crop_audit_df['dev_split'].isin(['train', 'val'])
].copy()
nonempty_target_df = target_audited_df[
    target_audited_df['target_voxels'] > 0
].copy()
estimable_coverage_df = nonempty_target_df[
    np.isfinite(nonempty_target_df['estimated_bbox_target_coverage'])
].copy()

crop_risk_mask = (
    target_audited_df['source_nonempty_became_cropped_empty'].fillna(False)
    | target_audited_df['target_touches_crop_boundary'].fillna(False)
    | target_audited_df['target_near_crop_boundary_under_4mm'].fillna(False)
    | target_audited_df['estimated_bbox_target_coverage'].fillna(1.0).lt(
        MINIMUM_ESTIMATED_BBOX_COVERAGE
    )
)
crop_risk_df = target_audited_df[crop_risk_mask].copy().sort_values([
    'source_nonempty_became_cropped_empty',
    'target_touches_crop_boundary',
    'minimum_target_edge_clearance_mm',
], ascending=[False, False, True])
atomic_write_csv(crop_risk_df, CROP_RISK_PATH)

cross_split_duplicate_count = (
    int(duplicate_audit_df['cross_split'].sum())
    if len(duplicate_audit_df)
    else 0
)
all_duplicate_count = len(duplicate_audit_df)

crop_checks = {
    **manifest_checks,
    'block1_official_metric_audit_passed': (
        block1_gate.get('audit_passed') is True
    ),
    'all_148_npz_hashes_match': len(crop_audit_df) == 148,
    'all_crop_geometry_inside_full_grid': bool(
        crop_audit_df['geometry_inside_full_grid'].all()
    ),
    'all_crop_affines_match_origins': bool(
        crop_audit_df['affine_translation_error'].le(1e-5).all()
    ),
    'all_125_train_and_validation_targets_audited': (
        len(target_audited_df) == 125
        and target_audited_df['target_array_read'].all()
    ),
    'test_target_arrays_not_read': (
        'test' not in target_arrays_read_splits
        and not crop_audit_df.loc[
            crop_audit_df['dev_split'] == 'test', 'target_array_read'
        ].any()
    ),
    'no_source_nonempty_target_became_cropped_empty': not bool(
        target_audited_df['source_nonempty_became_cropped_empty'].any()
    ),
    'no_target_touches_actual_crop_boundary': not bool(
        nonempty_target_df['target_touches_crop_boundary'].any()
    ),
    'estimated_bbox_target_coverage_at_least_0p995': (
        len(estimable_coverage_df) == len(nonempty_target_df)
        and bool(estimable_coverage_df['estimated_bbox_target_coverage'].ge(
            MINIMUM_ESTIMATED_BBOX_COVERAGE
        ).all())
    ),
    'stored_target_volume_metadata_matches_arrays': bool(
        target_audited_df['target_volume_metadata_error_ml']
        .fillna(0.0).le(1e-3).all()
    ),
    'no_exact_npz_or_input_duplicates': all_duplicate_count == 0,
    'no_cross_split_exact_input_duplicates': cross_split_duplicate_count == 0,
    **fold_checks,
    'fold_manifest_written': FOLD_MANIFEST_PATH.exists(),
    'training_not_performed': True,
    'model_training_not_authorized_by_this_block': True,
}
audit_passed = bool(all(crop_checks.values()))

fold_summary_records = fold_balance_df.to_dict('records')
minimum_edge_clearance = float(
    nonempty_target_df['minimum_target_edge_clearance_mm'].min()
)
minimum_estimated_coverage = float(
    estimable_coverage_df['estimated_bbox_target_coverage'].min()
) if len(estimable_coverage_df) else np.nan

atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'audit_version': AUDIT_VERSION,
        'audit_passed': audit_passed,
        'checks': crop_checks,
        'source_verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
        'source_verified_manifest_sha256': verified_manifest_sha256,
        'source_block1_gate_path': str(BLOCK1_GATE_PATH),
        'subjects': len(crop_audit_df),
        'split_counts': EXPECTED_SPLITS,
        'target_arrays_read_splits': sorted(target_arrays_read_splits),
        'test_target_arrays_read': False,
        'minimum_target_edge_clearance_mm': minimum_edge_clearance,
        'minimum_estimated_bbox_target_coverage': minimum_estimated_coverage,
        'near_boundary_subjects': int(
            nonempty_target_df['target_near_crop_boundary_under_4mm'].sum()
        ),
        'exact_duplicate_groups': all_duplicate_count,
        'cross_split_duplicate_groups': cross_split_duplicate_count,
        'number_of_development_folds': NUMBER_OF_FOLDS,
        'fold_seed': FOLD_SEED,
        'fold_summary': fold_summary_records,
        'fold_manifest_path': str(FOLD_MANIFEST_PATH),
        'fold_manifest_sha256': fold_manifest_sha256,
        'validation_policy': (
            'Previously observed validation cohort; lock for confirmation only. '
            'Use train-only five-fold out-of-fold development for subsequent changes.'
        ),
        'test_policy': 'sealed until final evaluation',
        'training_performed': False,
        'model_training_authorized': False,
        'study_trainable_parameters': 0,
        'next_block_authorized': audit_passed,
        'next_block': 'train-only out-of-fold baseline and sampling calibration',
    },
    AUDIT_GATE_PATH,
)

print('\nCrop integrity summary by split:')
display(
    crop_audit_df.groupby('dev_split', as_index=False).agg(
        subjects=('subject_id', 'count'),
        target_arrays_read=('target_array_read', 'sum'),
        mean_crop_reduction_fraction=('crop_reduction_fraction', 'mean'),
        minimum_target_edge_clearance_mm=(
            'minimum_target_edge_clearance_mm', 'min'
        ),
        near_boundary_subjects=(
            'target_near_crop_boundary_under_4mm', 'sum'
        ),
        target_boundary_touches=('target_touches_crop_boundary', 'sum'),
        minimum_estimated_bbox_coverage=(
            'estimated_bbox_target_coverage', 'min'
        ),
    )
)

print('\nTarget support versus actual crop-boundary interpretation:')
display(target_audited_df.groupby(
    ['dev_split', 'lesion_volume_bin'], as_index=False, dropna=False
).agg(
    subjects=('subject_id', 'count'),
    median_target_volume_ml=('target_volume_ml', 'median'),
    minimum_target_edge_clearance_mm=(
        'minimum_target_edge_clearance_mm', 'min'
    ),
    minimum_cropped_target_inside_support_fraction=(
        'cropped_target_inside_support_fraction', 'min'
    ),
    minimum_stored_full_target_support_coverage=(
        'stored_full_target_support_coverage', 'min'
    ),
    minimum_estimated_bbox_target_coverage=(
        'estimated_bbox_target_coverage', 'min'
    ),
))

print('\nDeterministic train-only five-fold balance:')
display(fold_balance_df)

print('\nFive-fold lesion-bin and atlas-mode counts:')
display(fold_strata_df[
    fold_strata_df['stratum'].isin(['lesion_volume_bin', 'atlas_mode'])
])

print('\nExploratory validation associations (not selection criteria):')
display(validation_association_df)

print('\nCrop-risk cases requiring attention:')
if len(crop_risk_df):
    display(crop_risk_df[[
        'subject_id', 'dev_split', 'lesion_volume_bin', 'target_volume_ml',
        'minimum_target_edge_clearance_mm',
        'target_touches_crop_boundary',
        'target_near_crop_boundary_under_4mm',
        'cropped_target_inside_support_fraction',
        'stored_full_target_support_coverage',
        'estimated_bbox_target_coverage',
    ]])
else:
    print('None.')

print('\nExact duplicate audit:')
if len(duplicate_audit_df):
    display(duplicate_audit_df)
else:
    print('No exact NPZ or input-image duplicates detected.')

print('\nBlock 2 checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed}
    for name, passed in crop_checks.items()
]))

if not audit_passed:
    failed = [name for name, passed in crop_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 CROP/FOLD INTEGRITY AUDIT FAILED. Do not begin new training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 CROP AND DEVELOPMENT-FOLD INTEGRITY AUDIT PASSED')
print('=' * 80)
print(f'Subjects audited:                  {len(crop_audit_df)}')
print('Train/validation targets audited:  125')
print('Test target arrays read:           False')
print(f'Minimum target edge clearance mm:  {minimum_edge_clearance:.3f}')
print(f'Minimum estimated bbox coverage:   {minimum_estimated_coverage:.6f}')
print(f'Crop-risk subjects:                {len(crop_risk_df)}')
print(f'Exact duplicate groups:            {all_duplicate_count}')
print(f'Development folds:                 {NUMBER_OF_FOLDS} (train only)')
print(f'Fold manifest SHA-256:             {fold_manifest_sha256}')
print('Validation policy:                 locked confirmation cohort')
print('Test policy:                       sealed')
print('Training performed:                False')
print('Model training authorized:         False')
print('Study trainable parameters:        0 (audit only)')
print(f'Crop audit:                        {CROP_AUDIT_PATH}')
print(f'Fold manifest:                     {FOLD_MANIFEST_PATH}')
print(f'Fold balance:                      {FOLD_BALANCE_PATH}')
print(f'Audit gate:                        {AUDIT_GATE_PATH}')
print('\nNext: check this output before the train-only OOF baseline block.')


In [ ]:
import sys
import os
import json
import math
import heapq
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            pip_name or import_name,
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
ensure_package('scipy')
ensure_package('matplotlib')

from nibabel.processing import resample_from_to
from scipy import ndimage
import matplotlib.pyplot as plt
from google.colab import drive
from IPython.display import display, Image as DisplayImage

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
BLOCK2_VERSION = 'v1_crop_boundary_and_train5fold_integrity'
DIAGNOSTIC_VERSION = 'v1_direct_source_to_final_grid_microlesion'
SUBJECT_ID = 'sub-stroke0171'

EXPECTED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_BLOCK2_FAILURE = 'no_source_nonempty_target_became_cropped_empty'

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata' / 'model_manifest_v1_trial0.csv'
)
BLOCK2_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / BLOCK2_VERSION
)
BLOCK2_GATE_PATH = BLOCK2_ROOT / 'crop_fold_integrity_gate.json'
BLOCK2_CROP_AUDIT_PATH = BLOCK2_ROOT / 'crop_boundary_integrity_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'target_integrity'
    / DIAGNOSTIC_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CASE_AUDIT_PATH = OUTPUT_ROOT / 'sub0171_microlesion_candidate_audit.csv'
THRESHOLD_AUDIT_PATH = OUTPUT_ROOT / 'sub0171_soft_threshold_audit.csv'
CANDIDATE_NIFTI_PATH = OUTPUT_ROOT / 'sub-stroke0171_candidate_binary.nii.gz'
DIRECT_SOFT_NIFTI_PATH = OUTPUT_ROOT / 'sub-stroke0171_direct_soft.nii.gz'
QA_PATH = OUTPUT_ROOT / 'sub-stroke0171_microlesion_candidate_qa.png'
DIAGNOSTIC_GATE_PATH = OUTPUT_ROOT / 'sub0171_microlesion_diagnostic_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_save_nifti(array, affine, path, dtype):
    temporary = path.with_name(path.name.replace('.nii.gz', '.tmp.nii.gz'))
    image = nib.Nifti1Image(np.asarray(array, dtype=dtype), affine)
    image.set_data_dtype(dtype)
    nib.save(image, temporary)
    os.replace(temporary, path)


def voxel_volume_mm3(affine):
    return float(abs(np.linalg.det(np.asarray(affine, dtype=np.float64)[:3, :3])))


def canonical_binary_image(path):
    image = nib.as_closest_canonical(nib.load(str(path)))
    values = np.asanyarray(image.dataobj)
    if not np.isfinite(values).all():
        raise RuntimeError('Source target contains non-finite values.')
    return image, values > 0


CONNECTIVITY_26 = np.ones((3, 3, 3), dtype=np.uint8)
NEIGHBOR_OFFSETS_26 = sorted(
    (dx, dy, dz)
    for dx in (-1, 0, 1)
    for dy in (-1, 0, 1)
    for dz in (-1, 0, 1)
    if (dx, dy, dz) != (0, 0, 0)
)


def component_count(mask):
    return int(ndimage.label(mask, structure=CONNECTIVITY_26)[1])


def grow_connected_candidate(score, affine, source_centroid_world, voxel_count):
    
    score = np.asarray(score, dtype=np.float64)
    shape = np.asarray(score.shape, dtype=np.int64)
    if score.ndim != 3 or voxel_count < 1:
        raise ValueError('Candidate score/voxel count is invalid.')

    positive_coordinates = np.argwhere(score > 0)
    if len(positive_coordinates):
        positive_scores = score[tuple(positive_coordinates.T)]
        maximum_score = float(positive_scores.max())
        seed_candidates = positive_coordinates[
            np.isclose(positive_scores, maximum_score, rtol=0.0, atol=0.0)
        ]
        seed = tuple(int(value) for value in seed_candidates[0])
    else:
        continuous_voxel = nib.affines.apply_affine(
            np.linalg.inv(affine), source_centroid_world
        )
        seed = tuple(
            int(value)
            for value in np.clip(np.rint(continuous_voxel), 0, shape - 1)
        )

    candidate = np.zeros(score.shape, dtype=bool)
    queued = np.zeros(score.shape, dtype=bool)
    queue = []

    def priority(coordinate):
        coordinate_array = np.asarray(coordinate, dtype=np.float64)
        world = nib.affines.apply_affine(affine, coordinate_array)
        distance = float(np.linalg.norm(world - source_centroid_world))
        flat_index = int(np.ravel_multi_index(coordinate, score.shape))
        return (-float(score[coordinate]), distance, flat_index, coordinate)

    heapq.heappush(queue, priority(seed))
    queued[seed] = True

    while queue and int(candidate.sum()) < voxel_count:
        _, _, _, coordinate = heapq.heappop(queue)
        if candidate[coordinate]:
            continue
        candidate[coordinate] = True
        for offset in NEIGHBOR_OFFSETS_26:
            neighbor = tuple(coordinate[axis] + offset[axis] for axis in range(3))
            if any(neighbor[axis] < 0 or neighbor[axis] >= shape[axis] for axis in range(3)):
                continue
            if not queued[neighbor]:
                heapq.heappush(queue, priority(neighbor))
                queued[neighbor] = True

    if int(candidate.sum()) != voxel_count:
        raise RuntimeError('Connected candidate growth did not reach requested volume.')
    return candidate, seed


def plane(array, axis, index):
    if axis == 0:
        return np.asarray(array[index, :, :]).T
    if axis == 1:
        return np.asarray(array[:, index, :]).T
    return np.asarray(array[:, :, index]).T






required_paths = [
    VERIFIED_MANIFEST_PATH,
    SOURCE_MANIFEST_PATH,
    BLOCK2_GATE_PATH,
    BLOCK2_CROP_AUDIT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 microlesion diagnostic prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
if verified_manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest changed. '
        f'Observed {verified_manifest_sha256}; expected {EXPECTED_MANIFEST_SHA256}.'
    )

block2_gate = json.loads(BLOCK2_GATE_PATH.read_text(encoding='utf-8'))
block2_checks = block2_gate.get('checks', {})
failed_block2_checks = sorted(
    name for name, passed in block2_checks.items() if passed is not True
)
if failed_block2_checks != [EXPECTED_BLOCK2_FAILURE]:
    raise RuntimeError(
        'Block 2 is not in the expected single-failure state. '
        f'Observed failed checks: {failed_block2_checks}'
    )
if block2_gate.get('test_target_arrays_read') is not False:
    raise RuntimeError('Block 2 reports test-target access.')

manifest_header = pd.read_csv(VERIFIED_MANIFEST_PATH, nrows=0).columns.tolist()
required_manifest_columns = {
    'subject_id', 'dev_split', 'vascufate_v2_npz_path', 'trial8_npz_sha256'
}
missing_manifest_columns = sorted(required_manifest_columns - set(manifest_header))
if missing_manifest_columns:
    raise RuntimeError(
        f'Verified manifest is missing columns: {missing_manifest_columns}'
    )

manifest_df = pd.read_csv(
    VERIFIED_MANIFEST_PATH,
    usecols=sorted(required_manifest_columns),
)
if len(manifest_df) != 148 or manifest_df['subject_id'].nunique() != 148:
    raise RuntimeError('Verified manifest subject count/uniqueness changed.')
if manifest_df['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
    raise RuntimeError('Verified manifest split counts changed.')

case_row = manifest_df.loc[manifest_df['subject_id'] == SUBJECT_ID]
if len(case_row) != 1:
    raise RuntimeError(f'Expected exactly one {SUBJECT_ID} manifest row.')
case_row = case_row.iloc[0]
if str(case_row['dev_split']) != 'train':
    raise RuntimeError(f'{SUBJECT_ID} is no longer a training subject.')

crop_audit_df = pd.read_csv(BLOCK2_CROP_AUDIT_PATH)
degenerate_df = crop_audit_df[
    crop_audit_df['source_nonempty_became_cropped_empty'].map(as_bool)
].copy()
if degenerate_df['subject_id'].tolist() != [SUBJECT_ID]:
    raise RuntimeError(
        'The Block 2 degenerate-target set changed: '
        f'{degenerate_df["subject_id"].tolist()}'
    )

source_header = pd.read_csv(SOURCE_MANIFEST_PATH, nrows=0).columns.tolist()
source_required = {'subject_id', 'dev_split', 'path_final_infarct_mask'}
source_missing = sorted(source_required - set(source_header))
if source_missing:
    raise RuntimeError(f'Source manifest is missing columns: {source_missing}')
source_df = pd.read_csv(SOURCE_MANIFEST_PATH, usecols=sorted(source_required))
source_row = source_df.loc[source_df['subject_id'] == SUBJECT_ID]
if len(source_row) != 1:
    raise RuntimeError(f'Expected one source row for {SUBJECT_ID}.')
source_row = source_row.iloc[0]
if str(source_row['dev_split']) != 'train':
    raise RuntimeError('Source and verified split labels disagree.')

npz_path = Path(str(case_row['vascufate_v2_npz_path']))
source_target_path = Path(str(source_row['path_final_infarct_mask']))
if not npz_path.exists() or not source_target_path.exists():
    raise FileNotFoundError(
        f'Missing case NPZ or source target: {npz_path}, {source_target_path}'
    )
npz_sha256 = file_sha256(npz_path)
if npz_sha256 != str(case_row['trial8_npz_sha256']):
    raise RuntimeError(f'{SUBJECT_ID}: frozen NPZ hash changed.')

print('Trial8 Block 2b upstream state verified.')
print(f'Frozen manifest SHA-256:            {verified_manifest_sha256}')
print(f'Expected Block 2 failure:           {EXPECTED_BLOCK2_FAILURE}')
print(f'Affected subject:                   {SUBJECT_ID} (train only)')
print('Test target arrays read:            False')
print('Study trainable parameters:         0 (diagnostic only)')
print('Frozen data modified:               False')






with np.load(npz_path, allow_pickle=False) as data:
    required_arrays = {
        'image', 'target_binary', 'target_soft', 'crop_mask', 'affine',
        'target_spacing', 'source_target_volume_ml', 'target_binary_volume_ml',
        'target_soft_volume_ml', 'binary_loss_weight',
    }
    missing_arrays = sorted(required_arrays - set(data.files))
    if missing_arrays:
        raise RuntimeError(f'{SUBJECT_ID}: missing arrays: {missing_arrays}')

    image = np.asarray(data['image'])
    stored_binary = np.asarray(data['target_binary']) > 0
    stored_soft = np.asarray(data['target_soft'], dtype=np.float32)
    crop_mask = np.asarray(data['crop_mask']) > 0
    affine = np.asarray(data['affine'], dtype=np.float64)
    spacing = np.asarray(data['target_spacing'], dtype=np.float64)
    source_volume_metadata_ml = scalar(data, 'source_target_volume_ml')
    stored_binary_volume_metadata_ml = scalar(data, 'target_binary_volume_ml')
    stored_soft_volume_metadata_ml = scalar(data, 'target_soft_volume_ml')
    stored_binary_loss_weight = scalar(data, 'binary_loss_weight')

if image.ndim != 4 or image.shape[0] != 15:
    raise RuntimeError('Stored image is not a 15-channel 3D volume.')
spatial_shape = tuple(int(value) for value in image.shape[1:])
for name, array in {
    'stored_binary': stored_binary,
    'stored_soft': stored_soft,
    'crop_mask': crop_mask,
}.items():
    if array.shape != spatial_shape:
        raise RuntimeError(f'{name} shape does not match the image grid.')
if not np.isfinite(stored_soft).all() or not np.isfinite(affine).all():
    raise RuntimeError('Stored target/affine contains non-finite values.')

source_image, source_binary = canonical_binary_image(source_target_path)
source_voxel_volume_mm3 = voxel_volume_mm3(source_image.affine)
output_voxel_volume_mm3 = voxel_volume_mm3(affine)
source_volume_recomputed_ml = float(
    source_binary.sum() * source_voxel_volume_mm3 / 1000.0
)
source_coordinates = np.argwhere(source_binary)
if len(source_coordinates) == 0:
    raise RuntimeError('Source target is unexpectedly empty.')
source_world = nib.affines.apply_affine(source_image.affine, source_coordinates)
source_centroid_world = source_world.mean(axis=0)
source_components = component_count(source_binary)

source_float_image = nib.Nifti1Image(
    source_binary.astype(np.float32), source_image.affine
)
destination = (spatial_shape, affine)
direct_nearest = (
    np.asanyarray(
        resample_from_to(source_float_image, destination, order=0, cval=0.0).dataobj
    ) > 0.5
)
direct_soft = np.clip(
    np.asanyarray(
        resample_from_to(source_float_image, destination, order=1, cval=0.0).dataobj
    ).astype(np.float32),
    0.0,
    1.0,
)
direct_half = direct_soft >= 0.5

requested_candidate_voxels = max(
    1,
    int(np.rint(source_volume_recomputed_ml * 1000.0 / output_voxel_volume_mm3)),
)
maximum_candidate_voxels = int(np.prod(spatial_shape))
requested_candidate_voxels = min(requested_candidate_voxels, maximum_candidate_voxels)
candidate_binary, candidate_seed = grow_connected_candidate(
    direct_soft,
    affine,
    source_centroid_world,
    requested_candidate_voxels,
)

voxel_volume_ml = output_voxel_volume_mm3 / 1000.0
stored_binary_volume_ml = float(stored_binary.sum() * voxel_volume_ml)
stored_soft_volume_ml = float(stored_soft.sum(dtype=np.float64) * voxel_volume_ml)
direct_nearest_volume_ml = float(direct_nearest.sum() * voxel_volume_ml)
direct_half_volume_ml = float(direct_half.sum() * voxel_volume_ml)
direct_soft_volume_ml = float(direct_soft.sum(dtype=np.float64) * voxel_volume_ml)
candidate_volume_ml = float(candidate_binary.sum() * voxel_volume_ml)
candidate_volume_error_ml = abs(candidate_volume_ml - source_volume_recomputed_ml)
candidate_inside_crop_mask_fraction = float(
    (candidate_binary & crop_mask).sum() / max(int(candidate_binary.sum()), 1)
)

threshold_rows = []
for threshold in [0.001, 0.005, 0.01, 0.025, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    threshold_mask = direct_soft >= threshold
    threshold_rows.append({
        'threshold': threshold,
        'voxels': int(threshold_mask.sum()),
        'volume_ml': float(threshold_mask.sum() * voxel_volume_ml),
        'components_26n': component_count(threshold_mask) if threshold_mask.any() else 0,
    })
threshold_df = pd.DataFrame(threshold_rows)
atomic_write_csv(threshold_df, THRESHOLD_AUDIT_PATH)

case_audit = {
    'subject_id': SUBJECT_ID,
    'dev_split': 'train',
    'npz_path': str(npz_path),
    'npz_sha256': npz_sha256,
    'source_target_path': str(source_target_path),
    'source_target_sha256': file_sha256(source_target_path),
    'source_shape': str(tuple(int(value) for value in source_binary.shape)),
    'final_shape': str(spatial_shape),
    'source_voxels': int(source_binary.sum()),
    'source_components_26n': source_components,
    'source_volume_metadata_ml': source_volume_metadata_ml,
    'source_volume_recomputed_ml': source_volume_recomputed_ml,
    'source_volume_metadata_error_ml': abs(
        source_volume_metadata_ml - source_volume_recomputed_ml
    ),
    'stored_binary_voxels': int(stored_binary.sum()),
    'stored_binary_volume_ml': stored_binary_volume_ml,
    'stored_binary_volume_metadata_ml': stored_binary_volume_metadata_ml,
    'stored_soft_nonzero_voxels': int((stored_soft > 0).sum()),
    'stored_soft_maximum': float(stored_soft.max()),
    'stored_soft_volume_ml': stored_soft_volume_ml,
    'stored_soft_volume_metadata_ml': stored_soft_volume_metadata_ml,
    'stored_binary_loss_weight': stored_binary_loss_weight,
    'direct_nearest_voxels': int(direct_nearest.sum()),
    'direct_nearest_volume_ml': direct_nearest_volume_ml,
    'direct_linear_nonzero_voxels': int((direct_soft > 0).sum()),
    'direct_linear_maximum': float(direct_soft.max()),
    'direct_linear_soft_volume_ml': direct_soft_volume_ml,
    'direct_half_threshold_voxels': int(direct_half.sum()),
    'direct_half_threshold_volume_ml': direct_half_volume_ml,
    'candidate_requested_voxels': requested_candidate_voxels,
    'candidate_voxels': int(candidate_binary.sum()),
    'candidate_volume_ml': candidate_volume_ml,
    'candidate_volume_error_ml': candidate_volume_error_ml,
    'candidate_components_26n': component_count(candidate_binary),
    'candidate_inside_crop_mask_fraction': candidate_inside_crop_mask_fraction,
    'candidate_seed': str(tuple(int(value) for value in candidate_seed)),
    'target_arrays_read_splits': 'train',
    'test_target_arrays_read': False,
    'frozen_data_modified': False,
    'study_trainable_parameters': 0,
}
case_audit_df = pd.DataFrame([case_audit])
atomic_write_csv(case_audit_df, CASE_AUDIT_PATH)

atomic_save_nifti(direct_soft, affine, DIRECT_SOFT_NIFTI_PATH, np.float32)
atomic_save_nifti(candidate_binary, affine, CANDIDATE_NIFTI_PATH, np.uint8)






candidate_coordinates = np.argwhere(candidate_binary)
candidate_center = np.rint(candidate_coordinates.mean(axis=0)).astype(int)
views = [
    ('sagittal', 0, int(candidate_center[0])),
    ('coronal', 1, int(candidate_center[1])),
    ('axial', 2, int(candidate_center[2])),
]

figure, axes = plt.subplots(3, 3, figsize=(14, 12), constrained_layout=True)
ncct = np.asarray(image[0], dtype=np.float32)
for row_index, (view_name, axis, slice_index) in enumerate(views):
    base = plane(ncct, axis, slice_index)
    stored = plane(stored_binary, axis, slice_index)
    soft = plane(direct_soft, axis, slice_index)
    candidate = plane(candidate_binary, axis, slice_index)

    axes[row_index, 0].imshow(base, cmap='gray', origin='lower', vmin=0, vmax=1)
    if stored.any():
        axes[row_index, 0].contour(stored, levels=[0.5], colors='cyan', linewidths=1.2)
    axes[row_index, 0].set_title(
        f'{view_name} {slice_index}: stored binary (cyan; expected empty)'
    )

    axes[row_index, 1].imshow(base, cmap='gray', origin='lower', vmin=0, vmax=1)
    masked_soft = np.ma.masked_where(soft <= 0, soft)
    axes[row_index, 1].imshow(
        masked_soft, cmap='viridis', origin='lower', alpha=0.85, vmin=0, vmax=1
    )
    axes[row_index, 1].set_title(
        f'{view_name} {slice_index}: direct source occupancy'
    )

    axes[row_index, 2].imshow(base, cmap='gray', origin='lower', vmin=0, vmax=1)
    masked_candidate = np.ma.masked_where(~candidate, candidate)
    axes[row_index, 2].imshow(
        masked_candidate, cmap='autumn', origin='lower', alpha=0.95, vmin=0, vmax=1
    )
    axes[row_index, 2].set_title(
        f'{view_name} {slice_index}: connected volume fallback (red)'
    )

    for column in range(3):
        axes[row_index, column].axis('off')

figure.suptitle(
    f'{SUBJECT_ID} microlesion reconstruction | '
    f'source={source_volume_recomputed_ml:.4f} mL, '
    f'candidate={candidate_volume_ml:.4f} mL',
    fontsize=14,
)
figure.savefig(QA_PATH, dpi=180, bbox_inches='tight')
plt.close(figure)






checks = {
    'verified_manifest_hash_matches': (
        verified_manifest_sha256 == EXPECTED_MANIFEST_SHA256
    ),
    'block2_has_only_expected_failure': (
        failed_block2_checks == [EXPECTED_BLOCK2_FAILURE]
    ),
    'exactly_one_degenerate_target': len(degenerate_df) == 1,
    'degenerate_target_is_sub_stroke0171': (
        degenerate_df['subject_id'].tolist() == [SUBJECT_ID]
    ),
    'affected_subject_is_train_only': str(case_row['dev_split']) == 'train',
    'source_target_is_nonempty': int(source_binary.sum()) > 0,
    'stored_binary_target_is_empty': int(stored_binary.sum()) == 0,
    'source_volume_metadata_matches_recomputed': (
        abs(source_volume_metadata_ml - source_volume_recomputed_ml) <= 1e-6
    ),
    'direct_source_resampling_has_support': bool((direct_soft > 0).any()),
    'candidate_is_nonempty': bool(candidate_binary.any()),
    'candidate_voxel_count_matches_request': (
        int(candidate_binary.sum()) == requested_candidate_voxels
    ),
    'candidate_is_one_connected_component': component_count(candidate_binary) == 1,
    'candidate_volume_error_within_half_output_voxel': (
        candidate_volume_error_ml <= (0.5 * voxel_volume_ml + 1e-12)
    ),
    'candidate_is_inside_final_array': True,
    'candidate_files_written': (
        CANDIDATE_NIFTI_PATH.exists()
        and DIRECT_SOFT_NIFTI_PATH.exists()
        and QA_PATH.exists()
    ),
    'test_target_arrays_not_read': True,
    'frozen_data_not_modified': True,
    'training_not_performed': True,
    'study_trainable_parameters_are_zero': True,
}
diagnostic_passed = bool(all(checks.values()))

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'diagnostic_version': DIAGNOSTIC_VERSION,
    'diagnostic_passed': diagnostic_passed,
    'checks': checks,
    'subject_id': SUBJECT_ID,
    'dev_split': 'train',
    'source_verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'source_verified_manifest_sha256': verified_manifest_sha256,
    'source_block2_gate_path': str(BLOCK2_GATE_PATH),
    'source_npz_path': str(npz_path),
    'source_npz_sha256': npz_sha256,
    'source_target_path': str(source_target_path),
    'source_target_sha256': file_sha256(source_target_path),
    'case_audit_path': str(CASE_AUDIT_PATH),
    'threshold_audit_path': str(THRESHOLD_AUDIT_PATH),
    'candidate_nifti_path': str(CANDIDATE_NIFTI_PATH),
    'direct_soft_nifti_path': str(DIRECT_SOFT_NIFTI_PATH),
    'visual_qa_path': str(QA_PATH),
    'candidate_policy': (
        'Only when a source lesion is nonempty and the stored final-grid binary '
        'is empty: resample the original source label directly to the final '
        'cropped grid, then grow one deterministic 26-connected component from '
        'the maximum occupancy seed to the nearest integer source-equivalent '
        'physical volume.'
    ),
    'test_target_arrays_read': False,
    'frozen_data_modified': False,
    'correction_committed': False,
    'training_performed': False,
    'model_training_authorized': False,
    'study_trainable_parameters': 0,
    'next_step': (
        'Check the candidate QA. If anatomically correct, run a separate atomic '
        'r3 target-correction and manifest/fold refreeze block.'
    ),
}
atomic_write_json(gate, DIAGNOSTIC_GATE_PATH)

print('\nMicrolesion reconstruction audit:')
display(case_audit_df.T.rename(columns={0: 'value'}))
print('\nDirect-linear occupancy threshold sensitivity:')
display(threshold_df)
print('\nDiagnostic checks:')
display(pd.DataFrame({
    'check': list(checks.keys()),
    'passed': list(checks.values()),
}))
display(DisplayImage(filename=str(QA_PATH)))

if not diagnostic_passed:
    failed = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 SUB-STROKE0171 MICROLESION DIAGNOSTIC FAILED. '
        'Do not alter the frozen dataset. Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 SUB-STROKE0171 MICROLESION DIAGNOSTIC PASSED')
print('=' * 80)
print(f'Source target volume:              {source_volume_recomputed_ml:.6f} mL')
print(f'Stored binary target volume:       {stored_binary_volume_ml:.6f} mL')
print(f'Stored soft target volume:         {stored_soft_volume_ml:.6f} mL')
print(f'Direct nearest target volume:      {direct_nearest_volume_ml:.6f} mL')
print(f'Direct linear soft volume:         {direct_soft_volume_ml:.6f} mL')
print(f'Candidate binary target volume:    {candidate_volume_ml:.6f} mL')
print(f'Candidate components:              {component_count(candidate_binary)}')
print(f'Candidate inside crop-mask frac.:  {candidate_inside_crop_mask_fraction:.6f}')
print('Test target arrays read:           False')
print('Frozen data modified:              False')
print('Model training authorized:         False')
print(f'Candidate target:                  {CANDIDATE_NIFTI_PATH}')
print(f'Visual QA:                         {QA_PATH}')
print(f'Diagnostic gate:                   {DIAGNOSTIC_GATE_PATH}')
print('\nStop here and check the red candidate against the source-occupancy panel.')


In [ ]:
import os
import json
import math
import hashlib
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
from scipy import ndimage
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
BLOCK2_VERSION = 'v1_crop_boundary_and_train5fold_integrity'
DIAGNOSTIC_VERSION = 'v1_direct_source_to_final_grid_microlesion'
POLICY_VERSION = 'v2_soft_only_subvoxel_train5fold'

EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SUBVOXEL_SUBJECT = 'sub-stroke0171'
EXPECTED_BLOCK2_FAILURE = 'no_source_nonempty_target_became_cropped_empty'
NUMBER_OF_FOLDS = 5
FOLD_SEED = 20260924
SOFT_MATCH_TOLERANCE = 1e-3

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
BLOCK2_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / BLOCK2_VERSION
)
BLOCK2_GATE_PATH = BLOCK2_ROOT / 'crop_fold_integrity_gate.json'
BLOCK2_CROP_AUDIT_PATH = BLOCK2_ROOT / 'crop_boundary_integrity_audit.csv'
BLOCK2_DUPLICATE_AUDIT_PATH = BLOCK2_ROOT / 'exact_input_duplicate_audit.csv'

DIAGNOSTIC_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'target_integrity'
    / DIAGNOSTIC_VERSION
)
DIAGNOSTIC_GATE_PATH = (
    DIAGNOSTIC_ROOT / 'sub0171_microlesion_diagnostic_gate.json'
)
DIAGNOSTIC_CASE_AUDIT_PATH = (
    DIAGNOSTIC_ROOT / 'sub0171_microlesion_candidate_audit.csv'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'target_integrity'
    / POLICY_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_POLICY_AUDIT_PATH = OUTPUT_ROOT / 'target_supervision_policy_audit.csv'
TARGET_POLICY_SUMMARY_PATH = OUTPUT_ROOT / 'target_supervision_policy_summary.csv'
FOLD_BALANCE_PATH = OUTPUT_ROOT / 'train5fold_balance_summary.csv'
FOLD_STRATA_PATH = OUTPUT_ROOT / 'train5fold_strata_counts.csv'
POLICY_GATE_PATH = OUTPUT_ROOT / 'target_policy_fold_refreeze_gate.json'

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v2_subvoxel_aware.csv'
)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def lesion_volume_bin(volume_ml):
    if volume_ml <= 0.0:
        return 'empty'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    if volume_ml < 75.0:
        return 'large'
    return 'very_large'


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


CONNECTIVITY_26 = np.ones((3, 3, 3), dtype=np.uint8)


def component_count(mask):
    if not np.asarray(mask).any():
        return 0
    return int(ndimage.label(mask, structure=CONNECTIVITY_26)[1])




atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'policy_version': POLICY_VERSION,
        'policy_frozen': False,
        'status': 'running_or_incomplete',
        'test_target_arrays_read': False,
        'frozen_npz_files_modified': False,
        'training_performed': False,
        'model_training_authorized': False,
    },
    POLICY_GATE_PATH,
)






required_paths = [
    VERIFIED_MANIFEST_PATH,
    BLOCK2_GATE_PATH,
    BLOCK2_CROP_AUDIT_PATH,
    BLOCK2_DUPLICATE_AUDIT_PATH,
    DIAGNOSTIC_GATE_PATH,
    DIAGNOSTIC_CASE_AUDIT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 subvoxel-policy prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )

block2_gate = json.loads(BLOCK2_GATE_PATH.read_text(encoding='utf-8'))
block2_checks = block2_gate.get('checks', {})
block2_failed_checks = sorted(
    name for name, passed in block2_checks.items() if passed is not True
)
if block2_failed_checks != [EXPECTED_BLOCK2_FAILURE]:
    raise RuntimeError(
        'Block 2 no longer has the expected single failure: '
        f'{block2_failed_checks}'
    )
if block2_gate.get('test_target_arrays_read') is not False:
    raise RuntimeError('Block 2 reports test-target access.')

diagnostic_gate = json.loads(
    DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8')
)
if diagnostic_gate.get('diagnostic_passed') is not True:
    raise RuntimeError('Sub-stroke0171 diagnostic gate is not passed.')
if diagnostic_gate.get('test_target_arrays_read') is not False:
    raise RuntimeError('Diagnostic gate reports test-target access.')
if diagnostic_gate.get('frozen_data_modified') is not False:
    raise RuntimeError('Diagnostic unexpectedly reports frozen-data modification.')

diagnostic_case_df = pd.read_csv(DIAGNOSTIC_CASE_AUDIT_PATH)
if len(diagnostic_case_df) != 1:
    raise RuntimeError('Expected one microlesion diagnostic row.')
diagnostic_case = diagnostic_case_df.iloc[0]
if str(diagnostic_case['subject_id']) != EXPECTED_SUBVOXEL_SUBJECT:
    raise RuntimeError('Unexpected microlesion diagnostic subject.')

source_components = int(diagnostic_case['source_components_26n'])
candidate_components = int(diagnostic_case['candidate_components_26n'])
hard_reconstruction_rejected = bool(
    source_components > 1 and candidate_components < source_components
)
if not hard_reconstruction_rejected:
    raise RuntimeError(
        'The expected component-loss evidence for rejecting the hard '
        'reconstruction is absent.'
    )

direct_soft_path = Path(str(diagnostic_gate['direct_soft_nifti_path']))
if not direct_soft_path.exists():
    raise FileNotFoundError(direct_soft_path)
direct_soft_image = nib.load(str(direct_soft_path))
direct_soft = np.asanyarray(direct_soft_image.dataobj).astype(np.float32)

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id', 'dev_split', 'vascufate_v2_npz_path', 'trial8_npz_sha256',
    'trial8_atlas_mode', 'trial8_atlas_enabled',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        f'Verified manifest is missing columns: {missing_manifest_columns}'
    )
manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError('Verified manifest no longer has 148 subjects.')
if manifest_df['subject_id'].nunique() != EXPECTED_SUBJECTS:
    raise RuntimeError('Verified manifest subject identifiers are not unique.')
if manifest_df['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
    raise RuntimeError('Verified manifest split counts changed.')

print('Trial8 Block 2c upstream state verified.')
print(f'Frozen manifest SHA-256:            {verified_manifest_sha256}')
print(f'Expected Block 2 failure:           {EXPECTED_BLOCK2_FAILURE}')
print(f'Hard candidate rejected:            {hard_reconstruction_rejected}')
print(f'Source/candidate components:        {source_components}/{candidate_components}')
print('Test target arrays authorized:      False')
print('Study trainable parameters:         0 (policy refreeze only)')






target_rows = []
target_arrays_read_splits = set()
for row in manifest_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if not npz_path.exists():
        raise FileNotFoundError(npz_path)
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')

    if split == 'test':
        target_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'target_array_read': False,
            'source_target_volume_ml': np.nan,
            'binary_voxels': np.nan,
            'binary_volume_ml': np.nan,
            'soft_sum': np.nan,
            'soft_volume_ml': np.nan,
            'soft_nonzero_voxels': np.nan,
            'binary_loss_weight': np.nan,
            'supervision_mode': 'sealed_test',
            'effective_target_volume_ml': np.nan,
            'effective_lesion_volume_bin': '',
            
            
            
            'sampling_positive': np.nan,
            'negative_sampling_exclusion_required': np.nan,
            'hard_metric_eligible': np.nan,
            'hard_target_available': np.nan,
            'soft_target_available': np.nan,
            'sampling_support_voxels': np.nan,
            'sampling_support_components_26n': np.nan,
            'sampling_support_inside_crop_fraction': np.nan,
            'all_finite': True,
        })
        continue

    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'target_binary', 'target_soft', 'binary_loss_weight', 'crop_mask',
            'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing target arrays: {missing}')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        source_volume_ml = scalar(data, 'source_target_volume_ml')
        binary_loss_weight = scalar(data, 'binary_loss_weight')
    target_arrays_read_splits.add(split)

    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: non-finite target metadata.')
    if soft.min() < -1e-5 or soft.max() > 1.0001:
        raise RuntimeError(f'{subject_id}: soft target is outside [0, 1].')
    if binary_loss_weight not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')

    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    binary_voxels = int(binary.sum())
    binary_volume_ml = float(binary_voxels * voxel_volume_ml)
    soft_sum = float(soft.sum(dtype=np.float64))
    soft_volume_ml = float(soft_sum * voxel_volume_ml)
    source_nonempty = source_volume_ml > 0.0
    hard_available = binary_voxels > 0
    soft_available = soft_sum > 0.0

    if source_nonempty and hard_available and soft_available:
        supervision_mode = 'hard_and_soft'
        effective_volume_ml = binary_volume_ml
        sampling_support = binary
        sampling_positive = True
        hard_metric_eligible = True
    elif (
        source_nonempty
        and not hard_available
        and soft_available
        and binary_loss_weight == 0.0
    ):
        supervision_mode = 'soft_only_subvoxel'
        effective_volume_ml = source_volume_ml
        sampling_support = soft > 0
        sampling_positive = True
        hard_metric_eligible = False
    elif not source_nonempty and not hard_available and not soft_available:
        supervision_mode = 'true_empty'
        effective_volume_ml = 0.0
        sampling_support = np.zeros_like(binary)
        sampling_positive = False
        hard_metric_eligible = True
    else:
        supervision_mode = 'invalid'
        effective_volume_ml = np.nan
        sampling_support = binary | (soft > 0)
        sampling_positive = source_nonempty
        hard_metric_eligible = False

    support_voxels = int(sampling_support.sum())
    support_inside_fraction = (
        1.0
        if support_voxels == 0
        else float((sampling_support & crop_mask).sum() / support_voxels)
    )

    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'target_array_read': True,
        'source_target_volume_ml': source_volume_ml,
        'binary_voxels': binary_voxels,
        'binary_volume_ml': binary_volume_ml,
        'soft_sum': soft_sum,
        'soft_volume_ml': soft_volume_ml,
        'soft_nonzero_voxels': int((soft > 0).sum()),
        'binary_loss_weight': binary_loss_weight,
        'supervision_mode': supervision_mode,
        'effective_target_volume_ml': effective_volume_ml,
        'effective_lesion_volume_bin': lesion_volume_bin(effective_volume_ml),
        'sampling_positive': sampling_positive,
        'negative_sampling_exclusion_required': sampling_positive,
        'hard_metric_eligible': hard_metric_eligible,
        'hard_target_available': hard_available,
        'soft_target_available': soft_available,
        'sampling_support_voxels': support_voxels,
        'sampling_support_components_26n': component_count(sampling_support),
        'sampling_support_inside_crop_fraction': support_inside_fraction,
        'all_finite': True,
    })

target_policy_df = pd.DataFrame(target_rows).sort_values('subject_id')

subvoxel_rows = target_policy_df[
    target_policy_df['supervision_mode'] == 'soft_only_subvoxel'
]
if subvoxel_rows['subject_id'].tolist() != [EXPECTED_SUBVOXEL_SUBJECT]:
    raise RuntimeError(
        'Unexpected soft-only subvoxel subjects: '
        f'{subvoxel_rows["subject_id"].tolist()}'
    )

subvoxel_manifest_row = manifest_df.loc[
    manifest_df['subject_id'] == EXPECTED_SUBVOXEL_SUBJECT
].iloc[0]
with np.load(
    Path(str(subvoxel_manifest_row['vascufate_v2_npz_path'])),
    allow_pickle=False,
) as data:
    stored_subvoxel_soft = np.asarray(data['target_soft'], dtype=np.float32)
    stored_subvoxel_affine = np.asarray(data['affine'], dtype=np.float64)

direct_soft_affine = np.asarray(direct_soft_image.affine, dtype=np.float64)
if direct_soft.shape != stored_subvoxel_soft.shape:
    raise RuntimeError('Direct/stored subvoxel soft-target shapes differ.')
soft_affine_error = float(
    np.max(np.abs(direct_soft_affine - stored_subvoxel_affine))
)
soft_maximum_absolute_error = float(
    np.max(np.abs(direct_soft - stored_subvoxel_soft))
)
soft_support_identical = bool(
    np.array_equal(direct_soft > 0, stored_subvoxel_soft > 0)
)

atomic_write_csv(target_policy_df, TARGET_POLICY_AUDIT_PATH)

policy_summary_df = (
    target_policy_df.groupby(
        ['dev_split', 'supervision_mode'], as_index=False, dropna=False
    )
    .agg(
        subjects=('subject_id', 'count'),
        sampling_positive=(
            'sampling_positive', lambda values: values.sum(min_count=1)
        ),
        hard_metric_eligible=(
            'hard_metric_eligible', lambda values: values.sum(min_count=1)
        ),
        minimum_effective_volume_ml=('effective_target_volume_ml', 'min'),
        maximum_effective_volume_ml=('effective_target_volume_ml', 'max'),
    )
)
atomic_write_csv(policy_summary_df, TARGET_POLICY_SUMMARY_PATH)






policy_columns = target_policy_df[[
    'subject_id',
    'target_array_read',
    'supervision_mode',
    'effective_target_volume_ml',
    'effective_lesion_volume_bin',
    'sampling_positive',
    'negative_sampling_exclusion_required',
    'hard_metric_eligible',
    'hard_target_available',
    'soft_target_available',
    'sampling_support_voxels',
    'sampling_support_components_26n',
]].copy()
policy_columns = policy_columns.rename(columns={
    column: f'trial8_{column}'
    for column in policy_columns.columns
    if column != 'subject_id'
})

target_policy_manifest_df = manifest_df.merge(
    policy_columns,
    on='subject_id',
    how='left',
    validate='one_to_one',
)
target_policy_manifest_df['trial8_target_policy_version'] = POLICY_VERSION
target_policy_manifest_df['trial8_target_policy_frozen'] = True
target_policy_manifest_df['trial8_model_training_authorized'] = False

atomic_write_csv(target_policy_manifest_df, TARGET_POLICY_MANIFEST_PATH)
target_policy_manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)






train_policy_df = target_policy_df[
    target_policy_df['dev_split'] == 'train'
].copy().sort_values('subject_id').reset_index(drop=True)
if len(train_policy_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Expected 102 train subjects for fold refreeze.')
if not train_policy_df['target_array_read'].all():
    raise RuntimeError('Train targets were not completely audited.')

center_candidates = [
    'center', 'center_id', 'site', 'site_id', 'institution', 'hospital',
    'acquisition_center', 'scanner_site', 'manufacturer',
]
center_columns = [
    column for column in center_candidates if column in manifest_df.columns
]

manifest_target_free = manifest_df[[
    'subject_id', 'dev_split', 'trial8_atlas_mode', 'trial8_atlas_enabled',
    *center_columns,
]].copy()
train_assignment_df = train_policy_df.merge(
    manifest_target_free,
    on=['subject_id', 'dev_split'],
    how='left',
    validate='one_to_one',
)

def center_metadata(record):
    parts = []
    for column in center_columns:
        value = record.get(column, '')
        if pd.notna(value) and str(value).strip():
            parts.append(f'{column}={value}')
    return '|'.join(parts) if parts else 'unavailable'


train_assignment_df['center_metadata'] = [
    center_metadata(record)
    for record in train_assignment_df.to_dict('records')
]

bin_frequency = Counter(
    train_assignment_df['effective_lesion_volume_bin'].astype(str)
)
mode_frequency = Counter(train_assignment_df['trial8_atlas_mode'].astype(str))
center_frequency = Counter(train_assignment_df['center_metadata'].astype(str))

assignment_records = train_assignment_df.to_dict('records')
assignment_records.sort(key=lambda record: (
    mode_frequency[str(record['trial8_atlas_mode'])],
    bin_frequency[str(record['effective_lesion_volume_bin'])],
    center_frequency[str(record['center_metadata'])],
    -float(record['effective_target_volume_ml']),
    str(record['subject_id']),
))

fold_size = Counter()
fold_bin_count = defaultdict(Counter)
fold_mode_count = defaultdict(Counter)
fold_center_count = defaultdict(Counter)
fold_log_volume = defaultdict(float)
fold_assignments = {}

for record in assignment_records:
    subject_id = str(record['subject_id'])
    lesion_bin = str(record['effective_lesion_volume_bin'])
    atlas_mode = str(record['trial8_atlas_mode'])
    center = str(record['center_metadata'])
    log_volume = math.log1p(float(record['effective_target_volume_ml']))
    rotation = int(hashlib.sha256(
        f'{FOLD_SEED}:{subject_id}'.encode('utf-8')
    ).hexdigest()[:8], 16) % NUMBER_OF_FOLDS
    fold_order = [
        (rotation + offset) % NUMBER_OF_FOLDS
        for offset in range(NUMBER_OF_FOLDS)
    ]
    selected_fold = min(
        fold_order,
        key=lambda fold: (
            fold_bin_count[fold][lesion_bin],
            fold_mode_count[fold][atlas_mode],
            fold_center_count[fold][center],
            fold_size[fold],
            fold_log_volume[fold],
            fold_order.index(fold),
        ),
    )
    fold_assignments[subject_id] = int(selected_fold)
    fold_size[selected_fold] += 1
    fold_bin_count[selected_fold][lesion_bin] += 1
    fold_mode_count[selected_fold][atlas_mode] += 1
    fold_center_count[selected_fold][center] += 1
    fold_log_volume[selected_fold] += log_volume

train_lookup = train_assignment_df.set_index('subject_id').to_dict('index')
manifest_lookup = manifest_target_free.set_index('subject_id').to_dict('index')
fold_rows = []
for row in manifest_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    if split == 'train':
        audit = train_lookup[subject_id]
        development_fold = int(fold_assignments[subject_id])
        development_role = 'cross_validation_pool'
        effective_bin = str(audit['effective_lesion_volume_bin'])
        effective_volume = float(audit['effective_target_volume_ml'])
        supervision_mode = str(audit['supervision_mode'])
        sampling_support_components = int(
            audit['sampling_support_components_26n']
        )
        center_value = str(audit['center_metadata'])
    elif split == 'val':
        development_fold = -1
        development_role = 'locked_validation_confirmation'
        effective_bin = ''
        effective_volume = np.nan
        supervision_mode = 'locked_validation'
        sampling_support_components = -1
        center_value = center_metadata(manifest_lookup[subject_id])
    else:
        development_fold = -1
        development_role = 'sealed_test'
        effective_bin = ''
        effective_volume = np.nan
        supervision_mode = 'sealed_test'
        sampling_support_components = -1
        center_value = center_metadata(manifest_lookup[subject_id])

    fold_rows.append({
        'subject_id': subject_id,
        'frozen_dev_split': split,
        'development_role': development_role,
        'development_fold': development_fold,
        'train_only_effective_lesion_volume_bin': effective_bin,
        'train_only_effective_target_volume_ml': effective_volume,
        'train_only_supervision_mode': supervision_mode,
        'train_only_sampling_support_components_26n': (
            sampling_support_components
        ),
        'atlas_mode': str(row['trial8_atlas_mode']),
        'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
        'center_metadata': center_value,
        'source_verified_manifest_sha256': verified_manifest_sha256,
        'target_policy_manifest_sha256': target_policy_manifest_sha256,
        'fold_seed': FOLD_SEED,
        'fold_policy': 'train_only_balanced_greedy_v2_subvoxel_aware',
    })

fold_manifest_df = pd.DataFrame(fold_rows).sort_values('subject_id')
train_fold_df = fold_manifest_df[
    fold_manifest_df['frozen_dev_split'] == 'train'
].copy()

fold_balance_df = (
    train_fold_df.groupby('development_fold', as_index=False)
    .agg(
        subjects=('subject_id', 'count'),
        effective_target_volume_mean_ml=(
            'train_only_effective_target_volume_ml', 'mean'
        ),
        effective_target_volume_median_ml=(
            'train_only_effective_target_volume_ml', 'median'
        ),
        effective_target_volume_sum_ml=(
            'train_only_effective_target_volume_ml', 'sum'
        ),
        atlas_enabled_subjects=('atlas_enabled', 'sum'),
        soft_only_subvoxel_subjects=(
            'train_only_supervision_mode',
            lambda values: int((values == 'soft_only_subvoxel').sum()),
        ),
    )
    .sort_values('development_fold')
    .reset_index(drop=True)
)

strata_rows = []
for stratum_name, column in [
    ('lesion_volume_bin', 'train_only_effective_lesion_volume_bin'),
    ('atlas_mode', 'atlas_mode'),
    ('center_metadata', 'center_metadata'),
    ('supervision_mode', 'train_only_supervision_mode'),
]:
    counts = (
        train_fold_df.groupby(['development_fold', column])
        .size()
        .unstack(fill_value=0)
        .reindex(range(NUMBER_OF_FOLDS), fill_value=0)
    )
    for category in counts.columns:
        for fold in range(NUMBER_OF_FOLDS):
            strata_rows.append({
                'stratum': stratum_name,
                'category': str(category),
                'development_fold': fold,
                'subjects': int(counts.loc[fold, category]),
                'category_total': int(counts[category].sum()),
                'maximum_minus_minimum': int(
                    counts[category].max() - counts[category].min()
                ),
            })
fold_strata_df = pd.DataFrame(strata_rows)

def stratum_balanced(stratum, minimum_total=1):
    subset = fold_strata_df[
        (fold_strata_df['stratum'] == stratum)
        & (fold_strata_df['category_total'] >= minimum_total)
    ]
    return bool(subset['maximum_minus_minimum'].le(1).all())


atomic_write_csv(fold_manifest_df, FOLD_MANIFEST_PATH)
atomic_write_csv(fold_balance_df, FOLD_BALANCE_PATH)
atomic_write_csv(fold_strata_df, FOLD_STRATA_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)






train_val_policy_df = target_policy_df[
    target_policy_df['dev_split'].isin(['train', 'val'])
]
invalid_policy_df = train_val_policy_df[
    train_val_policy_df['supervision_mode'] == 'invalid'
]
true_empty_df = train_val_policy_df[
    train_val_policy_df['supervision_mode'] == 'true_empty'
]
duplicate_audit_df = pd.read_csv(BLOCK2_DUPLICATE_AUDIT_PATH)

checks = {
    'verified_manifest_hash_matches': (
        verified_manifest_sha256 == EXPECTED_VERIFIED_MANIFEST_SHA256
    ),
    'block2_has_only_expected_failure': (
        block2_failed_checks == [EXPECTED_BLOCK2_FAILURE]
    ),
    'microlesion_diagnostic_passed': (
        diagnostic_gate.get('diagnostic_passed') is True
    ),
    'hard_reconstruction_rejected_for_component_loss': (
        hard_reconstruction_rejected
    ),
    'direct_and_stored_soft_shapes_match': (
        direct_soft.shape == stored_subvoxel_soft.shape
    ),
    'direct_and_stored_soft_affines_match': soft_affine_error <= 1e-5,
    'direct_and_stored_soft_support_match': soft_support_identical,
    'direct_and_stored_soft_values_match_within_1e_3': (
        soft_maximum_absolute_error <= SOFT_MATCH_TOLERANCE
    ),
    'all_125_train_validation_targets_audited': (
        len(train_val_policy_df) == 125
        and train_val_policy_df['target_array_read'].all()
    ),
    'test_target_arrays_not_read': (
        target_arrays_read_splits == {'train', 'val'}
        and not target_policy_df.loc[
            target_policy_df['dev_split'] == 'test', 'target_array_read'
        ].any()
    ),
    'no_invalid_supervision_modes': len(invalid_policy_df) == 0,
    'exactly_one_soft_only_subvoxel_subject': len(subvoxel_rows) == 1,
    'soft_only_subject_is_sub_stroke0171': (
        subvoxel_rows['subject_id'].tolist() == [EXPECTED_SUBVOXEL_SUBJECT]
    ),
    'subvoxel_case_is_sampling_positive': bool(
        subvoxel_rows['sampling_positive'].all()
    ),
    'subvoxel_case_excluded_from_hard_metrics': bool(
        (~subvoxel_rows['hard_metric_eligible']).all()
    ),
    'subvoxel_case_excluded_from_negative_sampling': bool(
        subvoxel_rows['negative_sampling_exclusion_required'].all()
    ),
    'subvoxel_sampling_support_preserves_two_components': bool(
        subvoxel_rows['sampling_support_components_26n'].eq(2).all()
    ),
    'subvoxel_sampling_support_inside_crop_mask': bool(
        subvoxel_rows['sampling_support_inside_crop_fraction'].ge(0.999999).all()
    ),
    'true_empty_cases_have_no_soft_or_hard_target': bool(
        (~true_empty_df['hard_target_available']).all()
        and (~true_empty_df['soft_target_available']).all()
    ),
    'no_exact_input_duplicates_from_block2': len(duplicate_audit_df) == 0,
    'target_policy_manifest_has_148_subjects': (
        len(target_policy_manifest_df) == EXPECTED_SUBJECTS
    ),
    'target_policy_manifest_subjects_unique': (
        target_policy_manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'target_policy_manifest_split_counts_match': (
        target_policy_manifest_df['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
    'exactly_102_train_subjects_assigned': len(train_fold_df) == 102,
    'every_train_subject_assigned_once': (
        train_fold_df['subject_id'].nunique() == 102
        and train_fold_df['development_fold'].between(0, 4).all()
    ),
    'validation_and_test_have_no_development_fold': fold_manifest_df.loc[
        fold_manifest_df['frozen_dev_split'].isin(['val', 'test']),
        'development_fold',
    ].eq(-1).all(),
    'fold_sizes_differ_by_at_most_one': (
        fold_balance_df['subjects'].max() - fold_balance_df['subjects'].min()
    ) <= 1,
    'lesion_volume_bins_differ_by_at_most_one': stratum_balanced(
        'lesion_volume_bin'
    ),
    'atlas_modes_with_five_or_more_cases_are_balanced': stratum_balanced(
        'atlas_mode', minimum_total=5
    ),
    'center_categories_with_five_or_more_cases_are_balanced': stratum_balanced(
        'center_metadata', minimum_total=5
    ),
    'common_supervision_modes_are_balanced': stratum_balanced(
        'supervision_mode', minimum_total=5
    ),
    'every_fold_contains_nonempty_supervision': bool(
        train_fold_df.groupby('development_fold')[
            'train_only_effective_target_volume_ml'
        ].apply(lambda values: (values > 0).any()).all()
    ),
    'frozen_npz_files_not_modified': True,
    'training_not_performed': True,
    'model_training_not_authorized_by_this_block': True,
    'study_trainable_parameters_are_zero': True,
}
policy_frozen = bool(all(checks.values()))

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'policy_version': POLICY_VERSION,
    'policy_frozen': policy_frozen,
    'checks': checks,
    'source_verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'source_verified_manifest_sha256': verified_manifest_sha256,
    'source_block2_gate_path': str(BLOCK2_GATE_PATH),
    'source_microlesion_diagnostic_gate_path': str(DIAGNOSTIC_GATE_PATH),
    'target_policy_manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'target_policy_manifest_sha256': target_policy_manifest_sha256,
    'fold_manifest_path': str(FOLD_MANIFEST_PATH),
    'fold_manifest_sha256': fold_manifest_sha256,
    'target_policy_audit_path': str(TARGET_POLICY_AUDIT_PATH),
    'fold_balance_path': str(FOLD_BALANCE_PATH),
    'fold_strata_path': str(FOLD_STRATA_PATH),
    'subvoxel_subjects': subvoxel_rows['subject_id'].tolist(),
    'subvoxel_policy': {
        'stored_binary_target': 'remain empty',
        'stored_soft_target': 'retain unchanged',
        'binary_loss_weight': 0.0,
        'positive_sampling_support': 'target_soft > 0',
        'negative_sampling': 'prohibited for soft-support overlap',
        'hard_metric_reporting': 'excluded',
        'fold_stratification_volume': 'source_target_volume_ml',
    },
    'hard_candidate_committed': False,
    'frozen_npz_files_modified': False,
    'test_target_arrays_read': False,
    'training_performed': False,
    'model_training_authorized': False,
    'study_trainable_parameters': 0,
    'next_block_authorized': policy_frozen,
    'next_block': (
        'five-fold loader/loss/sampler preflight with soft-only subvoxel '
        'positive handling and validation kept locked'
    ),
}
atomic_write_json(gate, POLICY_GATE_PATH)

print('\nTarget supervision policy summary:')
display(policy_summary_df)
print('\nSoft-only subvoxel subject:')
display(subvoxel_rows.T.rename(columns={subvoxel_rows.index[0]: 'value'}))
print('\nDirect versus stored soft-target agreement:')
display(pd.DataFrame([{
    'subject_id': EXPECTED_SUBVOXEL_SUBJECT,
    'shape': str(direct_soft.shape),
    'affine_maximum_error': soft_affine_error,
    'support_identical': soft_support_identical,
    'maximum_absolute_value_error': soft_maximum_absolute_error,
    'source_components': source_components,
    'rejected_hard_candidate_components': candidate_components,
}]))
print('\nRefrozen train-only fold balance:')
display(fold_balance_df)
print('\nRefrozen fold strata:')
display(fold_strata_df)
print('\nPolicy/fold checks:')
display(pd.DataFrame({
    'check': list(checks.keys()),
    'passed': list(checks.values()),
}))

if not policy_frozen:
    failed = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 SUBVOXEL TARGET POLICY/FOLD REFREEZE FAILED. '
        'Do not continue to training preflight. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 SUBVOXEL TARGET POLICY AND TRAIN-FOLD REFREEZE PASSED')
print('=' * 80)
print(f'Policy subjects:                    {len(target_policy_manifest_df)}')
print(f'Train/validation targets audited:   {len(train_val_policy_df)}')
print(f'Soft-only subvoxel subjects:        {subvoxel_rows["subject_id"].tolist()}')
print(f'True-empty train/validation cases:  {len(true_empty_df)}')
print(f'Direct/stored soft maximum error:   {soft_maximum_absolute_error:.8f}')
print(f'Hard candidate committed:           False')
print(f'Frozen NPZ files modified:          False')
print(f'Test target arrays read:            False')
print(f'Study trainable parameters:         0 (policy refreeze only)')
print(f'Target policy frozen:               {policy_frozen}')
print(f'Model training authorized:          False')
print(f'Target-policy manifest:             {TARGET_POLICY_MANIFEST_PATH}')
print(f'Target-policy manifest SHA-256:     {target_policy_manifest_sha256}')
print(f'Five-fold manifest:                 {FOLD_MANIFEST_PATH}')
print(f'Five-fold manifest SHA-256:         {fold_manifest_sha256}')
print(f'Policy gate:                        {POLICY_GATE_PATH}')
print(
    '\nNext: run the five-fold loader/loss/sampler preflight. It must center '
    'sub-stroke0171 from target_soft > 0, keep its hard loss disabled, and '
    'exclude it from hard-Dice aggregation and negative sampling.'
)


In [ ]:
import os
import json
import math
import hashlib
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
from scipy import ndimage
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
BLOCK2_VERSION = 'v1_crop_boundary_and_train5fold_integrity'
DIAGNOSTIC_VERSION = 'v1_direct_source_to_final_grid_microlesion'
POLICY_VERSION = 'v3_soft_only_subvoxel_train5fold_balanced'

EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_SUBVOXEL_SUBJECT = 'sub-stroke0171'
EXPECTED_BLOCK2_FAILURE = 'no_source_nonempty_target_became_cropped_empty'
NUMBER_OF_FOLDS = 5
FOLD_SEED = 20260924
SOFT_MATCH_TOLERANCE = 1e-3

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
BLOCK2_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / BLOCK2_VERSION
)
BLOCK2_GATE_PATH = BLOCK2_ROOT / 'crop_fold_integrity_gate.json'
BLOCK2_CROP_AUDIT_PATH = BLOCK2_ROOT / 'crop_boundary_integrity_audit.csv'
BLOCK2_DUPLICATE_AUDIT_PATH = BLOCK2_ROOT / 'exact_input_duplicate_audit.csv'

DIAGNOSTIC_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'target_integrity'
    / DIAGNOSTIC_VERSION
)
DIAGNOSTIC_GATE_PATH = (
    DIAGNOSTIC_ROOT / 'sub0171_microlesion_diagnostic_gate.json'
)
DIAGNOSTIC_CASE_AUDIT_PATH = (
    DIAGNOSTIC_ROOT / 'sub0171_microlesion_candidate_audit.csv'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'target_integrity'
    / POLICY_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_POLICY_AUDIT_PATH = OUTPUT_ROOT / 'target_supervision_policy_audit.csv'
TARGET_POLICY_SUMMARY_PATH = OUTPUT_ROOT / 'target_supervision_policy_summary.csv'
FOLD_BALANCE_PATH = OUTPUT_ROOT / 'train5fold_balance_summary.csv'
FOLD_STRATA_PATH = OUTPUT_ROOT / 'train5fold_strata_counts.csv'
POLICY_GATE_PATH = OUTPUT_ROOT / 'target_policy_fold_refreeze_gate.json'

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def lesion_volume_bin(volume_ml):
    if volume_ml <= 0.0:
        return 'empty'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    if volume_ml < 75.0:
        return 'large'
    return 'very_large'


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


CONNECTIVITY_26 = np.ones((3, 3, 3), dtype=np.uint8)


def component_count(mask):
    if not np.asarray(mask).any():
        return 0
    return int(ndimage.label(mask, structure=CONNECTIVITY_26)[1])




atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'policy_version': POLICY_VERSION,
        'policy_frozen': False,
        'status': 'running_or_incomplete',
        'test_target_arrays_read': False,
        'frozen_npz_files_modified': False,
        'training_performed': False,
        'model_training_authorized': False,
    },
    POLICY_GATE_PATH,
)






required_paths = [
    VERIFIED_MANIFEST_PATH,
    BLOCK2_GATE_PATH,
    BLOCK2_CROP_AUDIT_PATH,
    BLOCK2_DUPLICATE_AUDIT_PATH,
    DIAGNOSTIC_GATE_PATH,
    DIAGNOSTIC_CASE_AUDIT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 subvoxel-policy prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )

block2_gate = json.loads(BLOCK2_GATE_PATH.read_text(encoding='utf-8'))
block2_checks = block2_gate.get('checks', {})
block2_failed_checks = sorted(
    name for name, passed in block2_checks.items() if passed is not True
)
if block2_failed_checks != [EXPECTED_BLOCK2_FAILURE]:
    raise RuntimeError(
        'Block 2 no longer has the expected single failure: '
        f'{block2_failed_checks}'
    )
if block2_gate.get('test_target_arrays_read') is not False:
    raise RuntimeError('Block 2 reports test-target access.')

diagnostic_gate = json.loads(
    DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8')
)
if diagnostic_gate.get('diagnostic_passed') is not True:
    raise RuntimeError('Sub-stroke0171 diagnostic gate is not passed.')
if diagnostic_gate.get('test_target_arrays_read') is not False:
    raise RuntimeError('Diagnostic gate reports test-target access.')
if diagnostic_gate.get('frozen_data_modified') is not False:
    raise RuntimeError('Diagnostic unexpectedly reports frozen-data modification.')

diagnostic_case_df = pd.read_csv(DIAGNOSTIC_CASE_AUDIT_PATH)
if len(diagnostic_case_df) != 1:
    raise RuntimeError('Expected one microlesion diagnostic row.')
diagnostic_case = diagnostic_case_df.iloc[0]
if str(diagnostic_case['subject_id']) != EXPECTED_SUBVOXEL_SUBJECT:
    raise RuntimeError('Unexpected microlesion diagnostic subject.')

source_components = int(diagnostic_case['source_components_26n'])
candidate_components = int(diagnostic_case['candidate_components_26n'])
hard_reconstruction_rejected = bool(
    source_components > 1 and candidate_components < source_components
)
if not hard_reconstruction_rejected:
    raise RuntimeError(
        'The expected component-loss evidence for rejecting the hard '
        'reconstruction is absent.'
    )

direct_soft_path = Path(str(diagnostic_gate['direct_soft_nifti_path']))
if not direct_soft_path.exists():
    raise FileNotFoundError(direct_soft_path)
direct_soft_image = nib.load(str(direct_soft_path))
direct_soft = np.asanyarray(direct_soft_image.dataobj).astype(np.float32)

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id', 'dev_split', 'vascufate_v2_npz_path', 'trial8_npz_sha256',
    'trial8_atlas_mode', 'trial8_atlas_enabled',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        f'Verified manifest is missing columns: {missing_manifest_columns}'
    )
manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
if len(manifest_df) != EXPECTED_SUBJECTS:
    raise RuntimeError('Verified manifest no longer has 148 subjects.')
if manifest_df['subject_id'].nunique() != EXPECTED_SUBJECTS:
    raise RuntimeError('Verified manifest subject identifiers are not unique.')
if manifest_df['dev_split'].value_counts().to_dict() != EXPECTED_SPLITS:
    raise RuntimeError('Verified manifest split counts changed.')

print('Trial8 Block 2c upstream state verified.')
print(f'Frozen manifest SHA-256:            {verified_manifest_sha256}')
print(f'Expected Block 2 failure:           {EXPECTED_BLOCK2_FAILURE}')
print(f'Hard candidate rejected:            {hard_reconstruction_rejected}')
print(f'Source/candidate components:        {source_components}/{candidate_components}')
print('Test target arrays authorized:      False')
print('Study trainable parameters:         0 (policy refreeze only)')






target_rows = []
target_arrays_read_splits = set()
for row in manifest_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if not npz_path.exists():
        raise FileNotFoundError(npz_path)
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')

    if split == 'test':
        target_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'target_array_read': False,
            'source_target_volume_ml': np.nan,
            'binary_voxels': np.nan,
            'binary_volume_ml': np.nan,
            'soft_sum': np.nan,
            'soft_volume_ml': np.nan,
            'soft_nonzero_voxels': np.nan,
            'binary_loss_weight': np.nan,
            'supervision_mode': 'sealed_test',
            'effective_target_volume_ml': np.nan,
            'effective_lesion_volume_bin': '',
            
            
            
            'sampling_positive': np.nan,
            'negative_sampling_exclusion_required': np.nan,
            'hard_metric_eligible': np.nan,
            'hard_target_available': np.nan,
            'soft_target_available': np.nan,
            'sampling_support_voxels': np.nan,
            'sampling_support_components_26n': np.nan,
            'sampling_support_inside_crop_fraction': np.nan,
            'all_finite': True,
        })
        continue

    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'target_binary', 'target_soft', 'binary_loss_weight', 'crop_mask',
            'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing target arrays: {missing}')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float64)
        source_volume_ml = scalar(data, 'source_target_volume_ml')
        binary_loss_weight = scalar(data, 'binary_loss_weight')
    target_arrays_read_splits.add(split)

    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shape mismatch.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: non-finite target metadata.')
    if soft.min() < -1e-5 or soft.max() > 1.0001:
        raise RuntimeError(f'{subject_id}: soft target is outside [0, 1].')
    if binary_loss_weight not in {0.0, 1.0}:
        raise RuntimeError(f'{subject_id}: binary loss weight is invalid.')

    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    binary_voxels = int(binary.sum())
    binary_volume_ml = float(binary_voxels * voxel_volume_ml)
    soft_sum = float(soft.sum(dtype=np.float64))
    soft_volume_ml = float(soft_sum * voxel_volume_ml)
    source_nonempty = source_volume_ml > 0.0
    hard_available = binary_voxels > 0
    soft_available = soft_sum > 0.0

    if source_nonempty and hard_available and soft_available:
        supervision_mode = 'hard_and_soft'
        effective_volume_ml = binary_volume_ml
        sampling_support = binary
        sampling_positive = True
        hard_metric_eligible = True
    elif (
        source_nonempty
        and not hard_available
        and soft_available
        and binary_loss_weight == 0.0
    ):
        supervision_mode = 'soft_only_subvoxel'
        effective_volume_ml = source_volume_ml
        sampling_support = soft > 0
        sampling_positive = True
        hard_metric_eligible = False
    elif not source_nonempty and not hard_available and not soft_available:
        supervision_mode = 'true_empty'
        effective_volume_ml = 0.0
        sampling_support = np.zeros_like(binary)
        sampling_positive = False
        hard_metric_eligible = True
    else:
        supervision_mode = 'invalid'
        effective_volume_ml = np.nan
        sampling_support = binary | (soft > 0)
        sampling_positive = source_nonempty
        hard_metric_eligible = False

    support_voxels = int(sampling_support.sum())
    support_inside_fraction = (
        1.0
        if support_voxels == 0
        else float((sampling_support & crop_mask).sum() / support_voxels)
    )

    target_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'target_array_read': True,
        'source_target_volume_ml': source_volume_ml,
        'binary_voxels': binary_voxels,
        'binary_volume_ml': binary_volume_ml,
        'soft_sum': soft_sum,
        'soft_volume_ml': soft_volume_ml,
        'soft_nonzero_voxels': int((soft > 0).sum()),
        'binary_loss_weight': binary_loss_weight,
        'supervision_mode': supervision_mode,
        'effective_target_volume_ml': effective_volume_ml,
        'effective_lesion_volume_bin': lesion_volume_bin(effective_volume_ml),
        'sampling_positive': sampling_positive,
        'negative_sampling_exclusion_required': sampling_positive,
        'hard_metric_eligible': hard_metric_eligible,
        'hard_target_available': hard_available,
        'soft_target_available': soft_available,
        'sampling_support_voxels': support_voxels,
        'sampling_support_components_26n': component_count(sampling_support),
        'sampling_support_inside_crop_fraction': support_inside_fraction,
        'all_finite': True,
    })

target_policy_df = pd.DataFrame(target_rows).sort_values('subject_id')

subvoxel_rows = target_policy_df[
    target_policy_df['supervision_mode'] == 'soft_only_subvoxel'
]
if subvoxel_rows['subject_id'].tolist() != [EXPECTED_SUBVOXEL_SUBJECT]:
    raise RuntimeError(
        'Unexpected soft-only subvoxel subjects: '
        f'{subvoxel_rows["subject_id"].tolist()}'
    )

subvoxel_manifest_row = manifest_df.loc[
    manifest_df['subject_id'] == EXPECTED_SUBVOXEL_SUBJECT
].iloc[0]
with np.load(
    Path(str(subvoxel_manifest_row['vascufate_v2_npz_path'])),
    allow_pickle=False,
) as data:
    stored_subvoxel_soft = np.asarray(data['target_soft'], dtype=np.float32)
    stored_subvoxel_affine = np.asarray(data['affine'], dtype=np.float64)

direct_soft_affine = np.asarray(direct_soft_image.affine, dtype=np.float64)
if direct_soft.shape != stored_subvoxel_soft.shape:
    raise RuntimeError('Direct/stored subvoxel soft-target shapes differ.')
soft_affine_error = float(
    np.max(np.abs(direct_soft_affine - stored_subvoxel_affine))
)
soft_maximum_absolute_error = float(
    np.max(np.abs(direct_soft - stored_subvoxel_soft))
)
soft_support_identical = bool(
    np.array_equal(direct_soft > 0, stored_subvoxel_soft > 0)
)

atomic_write_csv(target_policy_df, TARGET_POLICY_AUDIT_PATH)

policy_summary_df = (
    target_policy_df.groupby(
        ['dev_split', 'supervision_mode'], as_index=False, dropna=False
    )
    .agg(
        subjects=('subject_id', 'count'),
        sampling_positive=(
            'sampling_positive', lambda values: values.sum(min_count=1)
        ),
        hard_metric_eligible=(
            'hard_metric_eligible', lambda values: values.sum(min_count=1)
        ),
        minimum_effective_volume_ml=('effective_target_volume_ml', 'min'),
        maximum_effective_volume_ml=('effective_target_volume_ml', 'max'),
    )
)
atomic_write_csv(policy_summary_df, TARGET_POLICY_SUMMARY_PATH)






policy_columns = target_policy_df[[
    'subject_id',
    'target_array_read',
    'supervision_mode',
    'effective_target_volume_ml',
    'effective_lesion_volume_bin',
    'sampling_positive',
    'negative_sampling_exclusion_required',
    'hard_metric_eligible',
    'hard_target_available',
    'soft_target_available',
    'sampling_support_voxels',
    'sampling_support_components_26n',
]].copy()
policy_columns = policy_columns.rename(columns={
    column: f'trial8_{column}'
    for column in policy_columns.columns
    if column != 'subject_id'
})

target_policy_manifest_df = manifest_df.merge(
    policy_columns,
    on='subject_id',
    how='left',
    validate='one_to_one',
)
target_policy_manifest_df['trial8_target_policy_version'] = POLICY_VERSION
target_policy_manifest_df['trial8_target_policy_frozen'] = True
target_policy_manifest_df['trial8_model_training_authorized'] = False

atomic_write_csv(target_policy_manifest_df, TARGET_POLICY_MANIFEST_PATH)
target_policy_manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)






train_policy_df = target_policy_df[
    target_policy_df['dev_split'] == 'train'
].copy().sort_values('subject_id').reset_index(drop=True)
if len(train_policy_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Expected 102 train subjects for fold refreeze.')
if not train_policy_df['target_array_read'].all():
    raise RuntimeError('Train targets were not completely audited.')

center_candidates = [
    'center', 'center_id', 'site', 'site_id', 'institution', 'hospital',
    'acquisition_center', 'scanner_site', 'manufacturer',
]
center_columns = [
    column for column in center_candidates if column in manifest_df.columns
]

manifest_target_free = manifest_df[[
    'subject_id', 'dev_split', 'trial8_atlas_mode', 'trial8_atlas_enabled',
    *center_columns,
]].copy()
train_assignment_df = train_policy_df.merge(
    manifest_target_free,
    on=['subject_id', 'dev_split'],
    how='left',
    validate='one_to_one',
)

def center_metadata(record):
    parts = []
    for column in center_columns:
        value = record.get(column, '')
        if pd.notna(value) and str(value).strip():
            parts.append(f'{column}={value}')
    return '|'.join(parts) if parts else 'unavailable'


train_assignment_df['center_metadata'] = [
    center_metadata(record)
    for record in train_assignment_df.to_dict('records')
]

bin_frequency = Counter(
    train_assignment_df['effective_lesion_volume_bin'].astype(str)
)
mode_frequency = Counter(train_assignment_df['trial8_atlas_mode'].astype(str))
center_frequency = Counter(train_assignment_df['center_metadata'].astype(str))

standard_records = [
    record
    for record in train_assignment_df.to_dict('records')
    if str(record['supervision_mode']) == 'hard_and_soft'
]
nonstandard_records = [
    record
    for record in train_assignment_df.to_dict('records')
    if str(record['supervision_mode']) != 'hard_and_soft'
]
if len(standard_records) != 100 or len(nonstandard_records) != 2:
    raise RuntimeError(
        'Expected 100 standard and 2 nonstandard training-supervision cases; '
        f'found {len(standard_records)} and {len(nonstandard_records)}.'
    )
if len(standard_records) % NUMBER_OF_FOLDS != 0:
    raise RuntimeError(
        'Standard supervision cases cannot be distributed equally across folds.'
    )
standard_fold_capacity = len(standard_records) // NUMBER_OF_FOLDS

standard_records.sort(key=lambda record: (
    mode_frequency[str(record['trial8_atlas_mode'])],
    bin_frequency[str(record['effective_lesion_volume_bin'])],
    center_frequency[str(record['center_metadata'])],
    -float(record['effective_target_volume_ml']),
    str(record['subject_id']),
))
nonstandard_records.sort(key=lambda record: (
    bin_frequency[str(record['effective_lesion_volume_bin'])],
    mode_frequency[str(record['trial8_atlas_mode'])],
    center_frequency[str(record['center_metadata'])],
    str(record['supervision_mode']),
    str(record['subject_id']),
))

fold_size = Counter()
fold_bin_count = defaultdict(Counter)
fold_mode_count = defaultdict(Counter)
fold_center_count = defaultdict(Counter)
fold_log_volume = defaultdict(float)
fold_nonstandard_count = Counter()
fold_assignments = {}

def assign_record(record, nonstandard):
    subject_id = str(record['subject_id'])
    lesion_bin = str(record['effective_lesion_volume_bin'])
    atlas_mode = str(record['trial8_atlas_mode'])
    center = str(record['center_metadata'])
    log_volume = math.log1p(float(record['effective_target_volume_ml']))
    rotation = int(hashlib.sha256(
        f'{FOLD_SEED}:{subject_id}'.encode('utf-8')
    ).hexdigest()[:8], 16) % NUMBER_OF_FOLDS
    fold_order = [
        (rotation + offset) % NUMBER_OF_FOLDS
        for offset in range(NUMBER_OF_FOLDS)
    ]
    if nonstandard:
        eligible_folds = fold_order
    else:
        eligible_folds = [
            fold for fold in fold_order
            if fold_size[fold] < standard_fold_capacity
        ]
        if not eligible_folds:
            raise RuntimeError(
                f'{subject_id}: no standard-supervision fold has capacity.'
            )

    def fold_priority(fold):
        if nonstandard:
            
            
            return (
                fold_nonstandard_count[fold],
                fold_bin_count[fold][lesion_bin],
                fold_mode_count[fold][atlas_mode],
                fold_center_count[fold][center],
                fold_size[fold],
                fold_log_volume[fold],
                fold_order.index(fold),
            )
        return (
            fold_bin_count[fold][lesion_bin],
            fold_mode_count[fold][atlas_mode],
            fold_center_count[fold][center],
            fold_size[fold],
            fold_log_volume[fold],
            fold_order.index(fold),
        )

    selected_fold = min(
        eligible_folds,
        key=fold_priority,
    )
    fold_assignments[subject_id] = int(selected_fold)
    fold_size[selected_fold] += 1
    fold_bin_count[selected_fold][lesion_bin] += 1
    fold_mode_count[selected_fold][atlas_mode] += 1
    fold_center_count[selected_fold][center] += 1
    fold_log_volume[selected_fold] += log_volume
    if nonstandard:
        fold_nonstandard_count[selected_fold] += 1




for record in standard_records:
    assign_record(record, nonstandard=False)
for record in nonstandard_records:
    assign_record(record, nonstandard=True)

train_lookup = train_assignment_df.set_index('subject_id').to_dict('index')
manifest_lookup = manifest_target_free.set_index('subject_id').to_dict('index')
fold_rows = []
for row in manifest_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    if split == 'train':
        audit = train_lookup[subject_id]
        development_fold = int(fold_assignments[subject_id])
        development_role = 'cross_validation_pool'
        effective_bin = str(audit['effective_lesion_volume_bin'])
        effective_volume = float(audit['effective_target_volume_ml'])
        supervision_mode = str(audit['supervision_mode'])
        sampling_support_components = int(
            audit['sampling_support_components_26n']
        )
        center_value = str(audit['center_metadata'])
    elif split == 'val':
        development_fold = -1
        development_role = 'locked_validation_confirmation'
        effective_bin = ''
        effective_volume = np.nan
        supervision_mode = 'locked_validation'
        sampling_support_components = -1
        center_value = center_metadata(manifest_lookup[subject_id])
    else:
        development_fold = -1
        development_role = 'sealed_test'
        effective_bin = ''
        effective_volume = np.nan
        supervision_mode = 'sealed_test'
        sampling_support_components = -1
        center_value = center_metadata(manifest_lookup[subject_id])

    fold_rows.append({
        'subject_id': subject_id,
        'frozen_dev_split': split,
        'development_role': development_role,
        'development_fold': development_fold,
        'train_only_effective_lesion_volume_bin': effective_bin,
        'train_only_effective_target_volume_ml': effective_volume,
        'train_only_supervision_mode': supervision_mode,
        'train_only_sampling_support_components_26n': (
            sampling_support_components
        ),
        'atlas_mode': str(row['trial8_atlas_mode']),
        'atlas_enabled': as_bool(row['trial8_atlas_enabled']),
        'center_metadata': center_value,
        'source_verified_manifest_sha256': verified_manifest_sha256,
        'target_policy_manifest_sha256': target_policy_manifest_sha256,
        'fold_seed': FOLD_SEED,
        'fold_policy': (
            'train_only_balanced_greedy_v3_standard20_then_'
            'nonstandard_spread'
        ),
    })

fold_manifest_df = pd.DataFrame(fold_rows).sort_values('subject_id')
train_fold_df = fold_manifest_df[
    fold_manifest_df['frozen_dev_split'] == 'train'
].copy()

fold_balance_df = (
    train_fold_df.groupby('development_fold', as_index=False)
    .agg(
        subjects=('subject_id', 'count'),
        effective_target_volume_mean_ml=(
            'train_only_effective_target_volume_ml', 'mean'
        ),
        effective_target_volume_median_ml=(
            'train_only_effective_target_volume_ml', 'median'
        ),
        effective_target_volume_sum_ml=(
            'train_only_effective_target_volume_ml', 'sum'
        ),
        atlas_enabled_subjects=('atlas_enabled', 'sum'),
        soft_only_subvoxel_subjects=(
            'train_only_supervision_mode',
            lambda values: int((values == 'soft_only_subvoxel').sum()),
        ),
        true_empty_subjects=(
            'train_only_supervision_mode',
            lambda values: int((values == 'true_empty').sum()),
        ),
    )
    .sort_values('development_fold')
    .reset_index(drop=True)
)

strata_rows = []
for stratum_name, column in [
    ('lesion_volume_bin', 'train_only_effective_lesion_volume_bin'),
    ('atlas_mode', 'atlas_mode'),
    ('center_metadata', 'center_metadata'),
    ('supervision_mode', 'train_only_supervision_mode'),
]:
    counts = (
        train_fold_df.groupby(['development_fold', column])
        .size()
        .unstack(fill_value=0)
        .reindex(range(NUMBER_OF_FOLDS), fill_value=0)
    )
    for category in counts.columns:
        for fold in range(NUMBER_OF_FOLDS):
            strata_rows.append({
                'stratum': stratum_name,
                'category': str(category),
                'development_fold': fold,
                'subjects': int(counts.loc[fold, category]),
                'category_total': int(counts[category].sum()),
                'maximum_minus_minimum': int(
                    counts[category].max() - counts[category].min()
                ),
            })
fold_strata_df = pd.DataFrame(strata_rows)

def stratum_balanced(stratum, minimum_total=1):
    subset = fold_strata_df[
        (fold_strata_df['stratum'] == stratum)
        & (fold_strata_df['category_total'] >= minimum_total)
    ]
    return bool(subset['maximum_minus_minimum'].le(1).all())


atomic_write_csv(fold_manifest_df, FOLD_MANIFEST_PATH)
atomic_write_csv(fold_balance_df, FOLD_BALANCE_PATH)
atomic_write_csv(fold_strata_df, FOLD_STRATA_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)






train_val_policy_df = target_policy_df[
    target_policy_df['dev_split'].isin(['train', 'val'])
]
invalid_policy_df = train_val_policy_df[
    train_val_policy_df['supervision_mode'] == 'invalid'
]
true_empty_df = train_val_policy_df[
    train_val_policy_df['supervision_mode'] == 'true_empty'
]
duplicate_audit_df = pd.read_csv(BLOCK2_DUPLICATE_AUDIT_PATH)

checks = {
    'verified_manifest_hash_matches': (
        verified_manifest_sha256 == EXPECTED_VERIFIED_MANIFEST_SHA256
    ),
    'block2_has_only_expected_failure': (
        block2_failed_checks == [EXPECTED_BLOCK2_FAILURE]
    ),
    'microlesion_diagnostic_passed': (
        diagnostic_gate.get('diagnostic_passed') is True
    ),
    'hard_reconstruction_rejected_for_component_loss': (
        hard_reconstruction_rejected
    ),
    'direct_and_stored_soft_shapes_match': (
        direct_soft.shape == stored_subvoxel_soft.shape
    ),
    'direct_and_stored_soft_affines_match': soft_affine_error <= 1e-5,
    'direct_and_stored_soft_support_match': soft_support_identical,
    'direct_and_stored_soft_values_match_within_1e_3': (
        soft_maximum_absolute_error <= SOFT_MATCH_TOLERANCE
    ),
    'all_125_train_validation_targets_audited': (
        len(train_val_policy_df) == 125
        and train_val_policy_df['target_array_read'].all()
    ),
    'test_target_arrays_not_read': (
        target_arrays_read_splits == {'train', 'val'}
        and not target_policy_df.loc[
            target_policy_df['dev_split'] == 'test', 'target_array_read'
        ].any()
    ),
    'no_invalid_supervision_modes': len(invalid_policy_df) == 0,
    'exactly_one_soft_only_subvoxel_subject': len(subvoxel_rows) == 1,
    'soft_only_subject_is_sub_stroke0171': (
        subvoxel_rows['subject_id'].tolist() == [EXPECTED_SUBVOXEL_SUBJECT]
    ),
    'subvoxel_case_is_sampling_positive': bool(
        subvoxel_rows['sampling_positive'].all()
    ),
    'subvoxel_case_excluded_from_hard_metrics': bool(
        (~subvoxel_rows['hard_metric_eligible']).all()
    ),
    'subvoxel_case_excluded_from_negative_sampling': bool(
        subvoxel_rows['negative_sampling_exclusion_required'].all()
    ),
    'subvoxel_sampling_support_preserves_two_components': bool(
        subvoxel_rows['sampling_support_components_26n'].eq(2).all()
    ),
    'subvoxel_sampling_support_inside_crop_mask': bool(
        subvoxel_rows['sampling_support_inside_crop_fraction'].ge(0.999999).all()
    ),
    'true_empty_cases_have_no_soft_or_hard_target': bool(
        (~true_empty_df['hard_target_available']).all()
        and (~true_empty_df['soft_target_available']).all()
    ),
    'no_exact_input_duplicates_from_block2': len(duplicate_audit_df) == 0,
    'target_policy_manifest_has_148_subjects': (
        len(target_policy_manifest_df) == EXPECTED_SUBJECTS
    ),
    'target_policy_manifest_subjects_unique': (
        target_policy_manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'target_policy_manifest_split_counts_match': (
        target_policy_manifest_df['dev_split'].value_counts().to_dict()
        == EXPECTED_SPLITS
    ),
    'exactly_102_train_subjects_assigned': len(train_fold_df) == 102,
    'every_train_subject_assigned_once': (
        train_fold_df['subject_id'].nunique() == 102
        and train_fold_df['development_fold'].between(0, 4).all()
    ),
    'exactly_20_hard_and_soft_subjects_per_fold': bool(
        train_fold_df[
            train_fold_df['train_only_supervision_mode'] == 'hard_and_soft'
        ].groupby('development_fold').size().reindex(
            range(NUMBER_OF_FOLDS), fill_value=0
        ).eq(20).all()
    ),
    'singleton_nonstandard_cases_are_in_distinct_folds': (
        train_fold_df[
            train_fold_df['train_only_supervision_mode']
            .isin(['soft_only_subvoxel', 'true_empty'])
        ]['development_fold'].nunique() == 2
    ),
    'validation_and_test_have_no_development_fold': fold_manifest_df.loc[
        fold_manifest_df['frozen_dev_split'].isin(['val', 'test']),
        'development_fold',
    ].eq(-1).all(),
    'fold_sizes_differ_by_at_most_one': (
        fold_balance_df['subjects'].max() - fold_balance_df['subjects'].min()
    ) <= 1,
    'lesion_volume_bins_differ_by_at_most_one': stratum_balanced(
        'lesion_volume_bin'
    ),
    'atlas_modes_with_five_or_more_cases_are_balanced': stratum_balanced(
        'atlas_mode', minimum_total=5
    ),
    'center_categories_with_five_or_more_cases_are_balanced': stratum_balanced(
        'center_metadata', minimum_total=5
    ),
    'common_supervision_modes_are_balanced': stratum_balanced(
        'supervision_mode', minimum_total=5
    ),
    'every_fold_contains_nonempty_supervision': bool(
        train_fold_df.groupby('development_fold')[
            'train_only_effective_target_volume_ml'
        ].apply(lambda values: (values > 0).any()).all()
    ),
    'frozen_npz_files_not_modified': True,
    'training_not_performed': True,
    'model_training_not_authorized_by_this_block': True,
    'study_trainable_parameters_are_zero': True,
}
policy_frozen = bool(all(checks.values()))

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'policy_version': POLICY_VERSION,
    'policy_frozen': policy_frozen,
    'checks': checks,
    'source_verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'source_verified_manifest_sha256': verified_manifest_sha256,
    'source_block2_gate_path': str(BLOCK2_GATE_PATH),
    'source_microlesion_diagnostic_gate_path': str(DIAGNOSTIC_GATE_PATH),
    'target_policy_manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'target_policy_manifest_sha256': target_policy_manifest_sha256,
    'fold_manifest_path': str(FOLD_MANIFEST_PATH),
    'fold_manifest_sha256': fold_manifest_sha256,
    'target_policy_audit_path': str(TARGET_POLICY_AUDIT_PATH),
    'fold_balance_path': str(FOLD_BALANCE_PATH),
    'fold_strata_path': str(FOLD_STRATA_PATH),
    'subvoxel_subjects': subvoxel_rows['subject_id'].tolist(),
    'subvoxel_policy': {
        'stored_binary_target': 'remain empty',
        'stored_soft_target': 'retain unchanged',
        'binary_loss_weight': 0.0,
        'positive_sampling_support': 'target_soft > 0',
        'negative_sampling': 'prohibited for soft-support overlap',
        'hard_metric_reporting': 'excluded',
        'fold_stratification_volume': 'source_target_volume_ml',
    },
    'hard_candidate_committed': False,
    'frozen_npz_files_modified': False,
    'test_target_arrays_read': False,
    'training_performed': False,
    'model_training_authorized': False,
    'study_trainable_parameters': 0,
    'next_block_authorized': policy_frozen,
    'next_block': (
        'five-fold loader/loss/sampler preflight with soft-only subvoxel '
        'positive handling and validation kept locked'
    ),
}
atomic_write_json(gate, POLICY_GATE_PATH)

print('\nTarget supervision policy summary:')
display(policy_summary_df)
print('\nSoft-only subvoxel subject:')
display(subvoxel_rows.T.rename(columns={subvoxel_rows.index[0]: 'value'}))
print('\nDirect versus stored soft-target agreement:')
display(pd.DataFrame([{
    'subject_id': EXPECTED_SUBVOXEL_SUBJECT,
    'shape': str(direct_soft.shape),
    'affine_maximum_error': soft_affine_error,
    'support_identical': soft_support_identical,
    'maximum_absolute_value_error': soft_maximum_absolute_error,
    'source_components': source_components,
    'rejected_hard_candidate_components': candidate_components,
}]))
print('\nRefrozen train-only fold balance:')
display(fold_balance_df)
print('\nRefrozen fold strata:')
display(fold_strata_df)
print('\nPolicy/fold checks:')
display(pd.DataFrame({
    'check': list(checks.keys()),
    'passed': list(checks.values()),
}))

if not policy_frozen:
    failed = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 SUBVOXEL TARGET POLICY/FOLD REFREEZE FAILED. '
        'Do not continue to training preflight. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 SUBVOXEL TARGET POLICY AND TRAIN-FOLD REFREEZE PASSED')
print('=' * 80)
print(f'Policy subjects:                    {len(target_policy_manifest_df)}')
print(f'Train/validation targets audited:   {len(train_val_policy_df)}')
print(f'Soft-only subvoxel subjects:        {subvoxel_rows["subject_id"].tolist()}')
print(f'True-empty train/validation cases:  {len(true_empty_df)}')
print(f'Direct/stored soft maximum error:   {soft_maximum_absolute_error:.8f}')
print(f'Hard candidate committed:           False')
print(f'Frozen NPZ files modified:          False')
print(f'Test target arrays read:            False')
print(f'Study trainable parameters:         0 (policy refreeze only)')
print(f'Target policy frozen:               {policy_frozen}')
print(f'Model training authorized:          False')
print(f'Target-policy manifest:             {TARGET_POLICY_MANIFEST_PATH}')
print(f'Target-policy manifest SHA-256:     {target_policy_manifest_sha256}')
print(f'Five-fold manifest:                 {FOLD_MANIFEST_PATH}')
print(f'Five-fold manifest SHA-256:         {fold_manifest_sha256}')
print(f'Policy gate:                        {POLICY_GATE_PATH}')
print(
    '\nNext: run the five-fold loader/loss/sampler preflight. It must center '
    'sub-stroke0171 from target_soft > 0, keep its hard loss disabled, and '
    'exclude it from hard-Dice aggregation and negative sampling.'
)


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')
scipy = ensure_package('scipy')

import torch.nn.functional as F
from scipy import ndimage
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
TARGET_POLICY_VERSION = 'v3_soft_only_subvoxel_train5fold_balanced'
PREFLIGHT_VERSION = 'v1_fivefold_softsubvoxel_sampler_loss'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_PARAMETERS = 15_052_628
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_GLOBAL_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

NUMBER_OF_FOLDS = 5
SEED = 20260924
PATCH_SIZE = (128, 128, 64)
MINIMUM_POSITIVE_PATCHES_PER_BIN = 20
MAXIMUM_SOFT_ONLY_PATCHES_PER_EPOCH = 1
MAXIMUM_NEGATIVE_CENTER_ATTEMPTS = 512
SCALER_EPSILON = 1e-6

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
TARGET_POLICY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'target_integrity'
    / TARGET_POLICY_VERSION
)
TARGET_POLICY_GATE_PATH = (
    TARGET_POLICY_ROOT / 'target_policy_fold_refreeze_gate.json'
)

GRAPH_VERIFICATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
)
GRAPH_SCHEMA_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)
GRAPH_VERIFICATION_GATE_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_verification_gate.json'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / 'v1_train_only_scaler_ragatv2_film'
)
MODEL_PREFLIGHT_GATE_PATH = (
    MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'
)
ARCHITECTURE_CONFIG_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
GLOBAL_SCALER_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_ROOT = OUTPUT_ROOT / 'fold_scalers'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SCALER_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_target_policy_recheck.csv'
SCALER_AUDIT_PATH = OUTPUT_ROOT / 'fold_scaler_audit.csv'
SAMPLING_AUDIT_PATH = OUTPUT_ROOT / 'fold_sampling_audit.csv'
FOLD_SCHEDULE_SUMMARY_PATH = OUTPUT_ROOT / 'fold_schedule_summary.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'loss_smoke_audit.csv'
TRAINING_POLICY_PATH = OUTPUT_ROOT / 'fivefold_training_policy.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'fivefold_loader_loss_sampler_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size):
    cropped = np.asarray(array)[slices]
    padding = []
    for observed, expected in zip(cropped.shape, patch_size):
        missing = max(int(expected) - int(observed), 0)
        lower = missing // 2
        padding.append((lower, missing - lower))
    if any(lower or upper for lower, upper in padding):
        cropped = np.pad(cropped, padding, mode='constant')
    output_slices = tuple(
        slice(0, min(cropped.shape[axis], patch_size[axis]))
        for axis in range(3)
    )
    return cropped[output_slices]


def pack_mask(mask):
    mask = np.asarray(mask, dtype=bool)
    return {
        'shape': tuple(int(value) for value in mask.shape),
        'packed': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def unpack_mask(record):
    voxel_count = int(np.prod(record['shape']))
    values = np.unpackbits(
        record['packed'], count=voxel_count, bitorder='little'
    )
    return values.reshape(record['shape']).astype(bool)


def subject_hash(subjects):
    return hashlib.sha256(
        '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    ).hexdigest()



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': PREFLIGHT_VERSION,
        'preflight_passed': False,
        'status': 'running_or_incomplete',
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'training_performed': False,
        'fold0_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
    },
    PREFLIGHT_GATE_PATH,
)






required_paths = [
    TARGET_POLICY_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    TARGET_POLICY_GATE_PATH,
    GRAPH_SCHEMA_PATH,
    GRAPH_VERIFICATION_GATE_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
    ARCHITECTURE_CONFIG_PATH,
    GLOBAL_SCALER_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 five-fold preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

target_policy_manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)
graph_schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
global_scaler_sha256 = file_sha256(GLOBAL_SCALER_PATH)
architecture_config_sha256 = file_sha256(ARCHITECTURE_CONFIG_PATH)

observed_expected_hashes = {
    'target_policy_manifest': (
        target_policy_manifest_sha256,
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256,
    ),
    'fold_manifest': (fold_manifest_sha256, EXPECTED_FOLD_MANIFEST_SHA256),
    'graph_schema': (graph_schema_sha256, EXPECTED_GRAPH_SCHEMA_SHA256),
    'global_scaler_schema_template': (
        global_scaler_sha256,
        EXPECTED_GLOBAL_SCALER_SHA256,
    ),
}
hash_errors = [
    f'{name}: observed {observed}, expected {expected}'
    for name, (observed, expected) in observed_expected_hashes.items()
    if observed != expected
]
if hash_errors:
    raise RuntimeError('Frozen hash mismatch:\n' + '\n'.join(hash_errors))

target_policy_gate = json.loads(
    TARGET_POLICY_GATE_PATH.read_text(encoding='utf-8')
)
graph_gate = json.loads(
    GRAPH_VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
architecture_config = json.loads(
    ARCHITECTURE_CONFIG_PATH.read_text(encoding='utf-8')
)

upstream_checks = {
    'target_policy_frozen': target_policy_gate.get('policy_frozen') is True,
    'target_policy_test_targets_sealed': (
        target_policy_gate.get('test_target_arrays_read') is False
    ),
    'target_policy_npz_files_unchanged': (
        target_policy_gate.get('frozen_npz_files_modified') is False
    ),
    'target_policy_manifest_hash_matches_gate': (
        target_policy_gate.get('target_policy_manifest_sha256')
        == target_policy_manifest_sha256
    ),
    'fold_manifest_hash_matches_gate': (
        target_policy_gate.get('fold_manifest_sha256') == fold_manifest_sha256
    ),
    'graph_verification_passed': graph_gate.get('verification_passed') is True,
    'graph_verification_target_free': graph_gate.get('target_arrays_read') is False,
    'model_preflight_passed': model_gate.get('preflight_passed') is True,
    'model_preflight_architecture_hash_matches': (
        model_gate.get('architecture_config_sha256')
        == architecture_config_sha256
    ),
    'model_preflight_scaler_hash_matches': (
        model_gate.get('scaler_sha256') == global_scaler_sha256
    ),
    'architecture_parameter_count_unchanged': (
        int(model_gate.get('total_parameters', -1)) == EXPECTED_PARAMETERS
    ),
    'architecture_config_parameter_count_unchanged': (
        int(architecture_config.get('total_parameters', -1))
        == EXPECTED_PARAMETERS
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream gates failed: ' + ', '.join(failed))

manifest_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id', 'dev_split', 'vascufate_v2_npz_path', 'trial8_npz_sha256',
    'trial8_atlas_enabled', 'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path', 'trial8_graph_sidecar_sha256',
    'trial8_supervision_mode', 'trial8_effective_target_volume_ml',
    'trial8_effective_lesion_volume_bin', 'trial8_sampling_positive',
    'trial8_negative_sampling_exclusion_required',
    'trial8_hard_metric_eligible', 'trial8_target_policy_frozen',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        f'Target-policy manifest is missing columns: {missing_manifest_columns}'
    )
required_fold_columns = {
    'subject_id', 'frozen_dev_split', 'development_fold',
    'development_role', 'train_only_supervision_mode',
    'train_only_effective_lesion_volume_bin',
    'target_policy_manifest_sha256',
}
missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
if missing_fold_columns:
    raise RuntimeError(f'Fold manifest is missing columns: {missing_fold_columns}')

manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
    raise RuntimeError('Target-policy and fold manifests have different subjects.')
fold_cross_checks = {
    'fold_splits_match_policy_manifest': fold_df['frozen_dev_split'].astype(
        str
    ).equals(manifest_df['dev_split'].astype(str)),
    'fold_policy_hash_matches_for_all_rows': fold_df[
        'target_policy_manifest_sha256'
    ].astype(str).eq(target_policy_manifest_sha256).all(),
    'train_fold_supervision_matches_policy': fold_df.loc[
        fold_df['frozen_dev_split'] == 'train',
        'train_only_supervision_mode',
    ].astype(str).reset_index(drop=True).equals(
        manifest_df.loc[
            manifest_df['dev_split'] == 'train',
            'trial8_supervision_mode',
        ].astype(str).reset_index(drop=True)
    ),
    'train_fold_volume_bins_match_policy': fold_df.loc[
        fold_df['frozen_dev_split'] == 'train',
        'train_only_effective_lesion_volume_bin',
    ].astype(str).reset_index(drop=True).equals(
        manifest_df.loc[
            manifest_df['dev_split'] == 'train',
            'trial8_effective_lesion_volume_bin',
        ].astype(str).reset_index(drop=True)
    ),
}
if not all(fold_cross_checks.values()):
    failed = [name for name, passed in fold_cross_checks.items() if not passed]
    raise RuntimeError('Fold-manifest cross-checks failed: ' + ', '.join(failed))
manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
manifest_df['development_role'] = fold_df['development_role'].astype(str)

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_are_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
    'policy_frozen_for_all': manifest_df['trial8_target_policy_frozen'].map(
        as_bool
    ).all(),
    'exactly_102_train_fold_assignments': (
        manifest_df.loc[manifest_df['dev_split'] == 'train', 'development_fold']
        .between(0, 4).all()
    ),
    'locked_validation_and_test_have_no_fold': manifest_df.loc[
        manifest_df['dev_split'].isin(['val', 'test']), 'development_fold'
    ].eq(-1).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Manifest checks failed: ' + ', '.join(failed))

train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != 102:
    raise RuntimeError('Expected exactly 102 train subjects.')

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    territory_edge_index = np.asarray(schema['territory_edge_index'], dtype=np.int64)
    territory_edge_type = np.asarray(schema['territory_edge_type'], dtype=np.int64)

with np.load(GLOBAL_SCALER_PATH, allow_pickle=False) as scaler_template:
    scaler_schema = {
        name: np.asarray(scaler_template[name])
        for name in [
            'territory_feature_names', 'vessel_feature_names',
            'vessel_source_indices', 'case_feature_names',
            'case_source_indices', 'case_scale_mask', 'clip_value',
        ]
    }

print('Trial8 five-fold preflight upstream gates passed.')
print(f'Policy subjects:                    {len(manifest_df)}')
print(f'Split counts:                       {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Target-policy manifest SHA-256:     {target_policy_manifest_sha256}')
print(f'Five-fold manifest SHA-256:         {fold_manifest_sha256}')
print(f'Architecture parameters:            {EXPECTED_PARAMETERS:,} (unchanged)')
print('Locked validation targets read:     False')
print('Test targets read:                  False')
print('Study trainable parameters:         0 (preflight only)')






territory_feature_names = [str(value) for value in scaler_schema[
    'territory_feature_names'
].tolist()]
vessel_feature_names = [str(value) for value in scaler_schema[
    'vessel_feature_names'
].tolist()]
case_feature_names = [str(value) for value in scaler_schema[
    'case_feature_names'
].tolist()]
vessel_source_indices = scaler_schema['vessel_source_indices'].astype(int)
case_source_indices = scaler_schema['case_source_indices'].astype(int)
case_scale_mask = scaler_schema['case_scale_mask'].astype(bool)
scaler_clip = float(np.asarray(scaler_schema['clip_value']).reshape(-1)[0])


def fit_feature_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    constants = np.zeros(len(feature_names), dtype=np.uint8)
    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid fold-scaler support for {feature_names[index]}.'
            )
        counts[index] = int(array.size)
        raw_std = float(array.std())
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
    return means, stds, counts, constants


scaler_rows = []
fold_scaler_paths = {}
fold_scaler_hashes = {}
for heldout_fold in range(NUMBER_OF_FOLDS):
    fit_df = train_df[train_df['development_fold'] != heldout_fold].copy()
    heldout_df = train_df[train_df['development_fold'] == heldout_fold].copy()
    expected_fit_count = 102 - len(heldout_df)

    territory_accumulators = [[] for _ in territory_feature_names]
    vessel_accumulators = [[] for _ in vessel_feature_names]
    case_accumulators = [[] for _ in case_feature_names]
    fit_subjects = []

    for row in fit_df.sort_values('subject_id').to_dict('records'):
        subject_id = str(row['subject_id'])
        sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
        if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
            raise RuntimeError(f'{subject_id}: graph-sidecar hash changed.')
        with np.load(sidecar_path, allow_pickle=False) as data:
            forbidden = {
                'target_binary', 'target_soft', 'binary_loss_weight',
                'source_target_volume_ml', 'target_binary_volume_ml',
                'target_soft_volume_ml',
            }
            if forbidden & set(data.files):
                raise RuntimeError(f'{subject_id}: target found in graph sidecar.')
            node_features = np.asarray(data['node_features'], dtype=np.float32)
            node_feature_valid = np.asarray(data['node_feature_valid']) > 0
            node_valid = np.asarray(data['node_valid']) > 0
            vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
            vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
            vessel_presence = np.asarray(data['vessel_presence']) > 0
            case_features = np.asarray(data['case_features'], dtype=np.float32)

        expected_node_shape = (30, len(territory_feature_names))
        if node_features.shape != expected_node_shape:
            raise RuntimeError(
                f'{subject_id}: node feature shape {node_features.shape} != '
                f'{expected_node_shape}.'
            )
        if node_feature_valid.shape != node_features.shape:
            raise RuntimeError(f'{subject_id}: node feature-valid shape differs.')
        if node_valid.shape != (30,):
            raise RuntimeError(f'{subject_id}: node-valid shape differs.')
        if vessel_features.ndim != 2 or vessel_features.shape[0] != 13:
            raise RuntimeError(f'{subject_id}: vessel feature shape differs.')
        if vessel_feature_valid.shape != vessel_features.shape:
            raise RuntimeError(f'{subject_id}: vessel feature-valid shape differs.')
        if vessel_presence.shape != (13,):
            raise RuntimeError(f'{subject_id}: vessel-presence shape differs.')
        if vessel_source_indices.max(initial=-1) >= vessel_features.shape[1]:
            raise RuntimeError(f'{subject_id}: vessel source index is invalid.')
        if case_source_indices.max(initial=-1) >= case_features.size:
            raise RuntimeError(f'{subject_id}: case source index is invalid.')
        if not all(
            np.isfinite(array).all()
            for array in [node_features, vessel_features, case_features]
        ):
            raise RuntimeError(f'{subject_id}: non-finite graph feature found.')

        for feature_index in range(len(territory_feature_names)):
            valid = node_valid & node_feature_valid[:, feature_index]
            territory_accumulators[feature_index].extend(
                node_features[valid, feature_index].astype(float).tolist()
            )
        for model_index, source_index in enumerate(vessel_source_indices):
            valid = vessel_presence & vessel_feature_valid[:, source_index]
            vessel_accumulators[model_index].extend(
                vessel_features[valid, source_index].astype(float).tolist()
            )
        for model_index, source_index in enumerate(case_source_indices):
            case_accumulators[model_index].append(
                float(case_features[source_index])
            )
        fit_subjects.append(subject_id)

    territory_mean, territory_std, territory_count, territory_constant = (
        fit_feature_scaler(territory_accumulators, territory_feature_names)
    )
    vessel_mean, vessel_std, vessel_count, vessel_constant = fit_feature_scaler(
        vessel_accumulators, vessel_feature_names
    )
    case_mean, case_std, case_count, case_constant = fit_feature_scaler(
        case_accumulators, case_feature_names, scale_mask=case_scale_mask
    )

    scaler_path = SCALER_ROOT / f'fold_{heldout_fold}_graph_scaler.npz'
    fit_hash = subject_hash(fit_subjects)
    heldout_subjects = heldout_df['subject_id'].astype(str).tolist()
    atomic_write_npz(
        scaler_path,
        territory_feature_names=np.asarray(territory_feature_names),
        territory_mean=territory_mean,
        territory_std=territory_std,
        territory_count=territory_count,
        vessel_feature_names=np.asarray(vessel_feature_names),
        vessel_source_indices=vessel_source_indices.astype(np.int16),
        vessel_mean=vessel_mean,
        vessel_std=vessel_std,
        vessel_count=vessel_count,
        case_feature_names=np.asarray(case_feature_names),
        case_source_indices=case_source_indices.astype(np.int16),
        case_scale_mask=case_scale_mask.astype(np.uint8),
        case_mean=case_mean,
        case_std=case_std,
        case_count=case_count,
        clip_value=np.asarray(scaler_clip, dtype=np.float32),
        heldout_fold=np.asarray(heldout_fold, dtype=np.int8),
        fit_subject_count=np.asarray(len(fit_subjects), dtype=np.int16),
        fit_subject_hash=np.asarray(fit_hash),
        heldout_subject_hash=np.asarray(subject_hash(heldout_subjects)),
        target_arrays_read=np.asarray(False, dtype=np.uint8),
    )
    scaler_sha256 = file_sha256(scaler_path)
    fold_scaler_paths[heldout_fold] = str(scaler_path)
    fold_scaler_hashes[heldout_fold] = scaler_sha256
    scaler_rows.append({
        'heldout_fold': heldout_fold,
        'fit_subjects': len(fit_subjects),
        'expected_fit_subjects': expected_fit_count,
        'heldout_subjects': len(heldout_subjects),
        'fit_and_heldout_disjoint': not bool(
            set(fit_subjects) & set(heldout_subjects)
        ),
        'fit_subject_hash': fit_hash,
        'heldout_subject_hash': subject_hash(heldout_subjects),
        'territory_features_supported': bool((territory_count > 0).all()),
        'vessel_features_supported': bool((vessel_count > 0).all()),
        'case_features_supported': bool((case_count > 0).all()),
        'territory_scaled_features_nonconstant': not bool(
            territory_constant.any()
        ),
        'vessel_scaled_features_nonconstant': not bool(vessel_constant.any()),
        'case_scaled_features_nonconstant': not bool(
            case_constant[case_scale_mask].any()
        ),
        'all_statistics_finite': all(
            np.isfinite(array).all()
            for array in [
                territory_mean, territory_std, vessel_mean, vessel_std,
                case_mean, case_std,
            ]
        ),
        'all_scaled_stds_positive': bool(
            (territory_std > 0).all()
            and (vessel_std > 0).all()
            and (case_std[case_scale_mask] > 0).all()
        ),
        'heldout_sidecars_read_for_scaler': False,
        'target_arrays_read_for_scaler': False,
        'scaler_path': str(scaler_path),
        'scaler_sha256': scaler_sha256,
    })

scaler_audit_df = pd.DataFrame(scaler_rows)
atomic_write_csv(scaler_audit_df, SCALER_AUDIT_PATH)






target_rows = []
mask_records = {}
train_by_subject = {
    str(row['subject_id']): row for row in train_df.to_dict('records')
}

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'target_binary', 'target_soft', 'binary_loss_weight', 'crop_mask',
            'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing target arrays: {missing}')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        binary_loss_weight = scalar(data, 'binary_loss_weight')
        source_volume_ml = scalar(data, 'source_target_volume_ml')
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if not np.isfinite(soft).all():
        raise RuntimeError(f'{subject_id}: soft target is non-finite.')

    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        sampling_support = binary
        expected_hard_weight = 1.0
    elif supervision_mode == 'soft_only_subvoxel':
        sampling_support = soft > 0
        expected_hard_weight = 0.0
    elif supervision_mode == 'true_empty':
        sampling_support = np.zeros_like(binary)
        expected_hard_weight = 1.0
    else:
        raise RuntimeError(f'{subject_id}: invalid train supervision mode.')

    exclusion_support = binary | (soft > 0)
    support_voxels = int(sampling_support.sum())
    positive_expected = as_bool(row['trial8_sampling_positive'])
    if positive_expected != (support_voxels > 0):
        raise RuntimeError(f'{subject_id}: positive-sampling policy mismatch.')
    if not math.isclose(binary_loss_weight, expected_hard_weight):
        raise RuntimeError(f'{subject_id}: hard-loss policy mismatch.')

    mask_records[subject_id] = {
        'sampling_support': pack_mask(sampling_support),
        'exclusion_support': pack_mask(exclusion_support),
        'crop_mask': pack_mask(crop_mask),
    }
    target_rows.append({
        'subject_id': subject_id,
        'development_fold': int(row['development_fold']),
        'supervision_mode': supervision_mode,
        'effective_lesion_volume_bin': str(
            row['trial8_effective_lesion_volume_bin']
        ),
        'effective_target_volume_ml': float(
            row['trial8_effective_target_volume_ml']
        ),
        'sampling_positive': positive_expected,
        'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
        'binary_loss_weight': binary_loss_weight,
        'source_target_volume_ml': source_volume_ml,
        'binary_voxels': int(binary.sum()),
        'soft_nonzero_voxels': int((soft > 0).sum()),
        'soft_sum': float(soft.sum(dtype=np.float64)),
        'sampling_support_voxels': support_voxels,
        'exclusion_support_voxels': int(exclusion_support.sum()),
        'support_inside_crop_fraction': (
            1.0
            if support_voxels == 0
            else float((sampling_support & crop_mask).sum() / support_voxels)
        ),
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_by_subject = target_audit_df.set_index('subject_id').to_dict('index')






region_map_cache = {}


def load_region_map(subject_id):
    if subject_id in region_map_cache:
        return region_map_cache[subject_id]
    row = train_by_subject[subject_id]
    shape = mask_records[subject_id]['sampling_support']['shape']
    if not as_bool(row['trial8_atlas_enabled']):
        region = np.zeros(shape, dtype=np.uint8)
    else:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        image = nib.load(str(atlas_path))
        region = np.asanyarray(image.dataobj)
        if region.ndim == 4 and region.shape[-1] == 1:
            region = region[..., 0]
        region = np.rint(region).astype(np.uint8)
        if region.shape != shape:
            raise RuntimeError(f'{subject_id}: atlas/target shape mismatch.')
        if region.min() < 0 or region.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    region_map_cache[subject_id] = region
    return region


def topology_candidate_labels(support, region_map):
    lesion_labels = set(
        int(value)
        for value in np.unique(region_map[support])
        if 1 <= int(value) <= 30
    )
    if not lesion_labels:
        return [], {}
    relation_priority = {2: 0, 1: 1, 3: 2}
    relation_by_label = {}
    for edge_index in range(territory_edge_index.shape[1]):
        source = int(territory_edge_index[0, edge_index]) + 1
        destination = int(territory_edge_index[1, edge_index]) + 1
        relation = int(territory_edge_type[edge_index])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in relation_priority
        ):
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    ordered = sorted(
        relation_by_label,
        key=lambda label: (relation_priority[relation_by_label[label]], label),
    )
    return ordered, relation_by_label


RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def choose_positive_center(subject_id, rng, component_index=None):
    support = unpack_mask(mask_records[subject_id]['sampling_support'])
    if not support.any():
        raise RuntimeError(f'{subject_id}: positive case has no support.')
    if component_index is None:
        coordinates = np.argwhere(support)
    else:
        labels, count = ndimage_label_26n(support)
        if count < 1:
            raise RuntimeError(f'{subject_id}: support has no component.')
        selected_label = 1 + (int(component_index) % count)
        coordinates = np.argwhere(labels == selected_label)
    center = coordinates[int(rng.integers(0, len(coordinates)))].astype(np.int64)
    jitter = np.asarray([
        int(rng.integers(-12, 13)),
        int(rng.integers(-12, 13)),
        int(rng.integers(-4, 5)),
    ])
    center = np.clip(center + jitter, 0, np.asarray(support.shape) - 1)
    slices = patch_slices_from_center(support.shape, center, PATCH_SIZE)
    patch_support_voxels = int(support[slices].sum())
    return center, patch_support_voxels


def ndimage_label_26n(mask):
    return ndimage.label(mask, structure=np.ones((3, 3, 3), dtype=np.uint8))


def shuffled_coordinates(mask, rng, maximum=MAXIMUM_NEGATIVE_CENTER_ATTEMPTS):
    flat = np.flatnonzero(mask)
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > maximum:
        flat = rng.choice(flat, size=maximum, replace=False)
    else:
        flat = flat.copy()
        rng.shuffle(flat)
    return np.column_stack(np.unravel_index(flat, mask.shape)).astype(np.int64)


def first_zero_support_center(exclusion_support, candidate_mask, rng):
    coordinates = shuffled_coordinates(candidate_mask, rng)
    for center in coordinates:
        slices = patch_slices_from_center(
            exclusion_support.shape, center, PATCH_SIZE
        )
        if not exclusion_support[slices].any():
            return center, slices
    return None, None


def choose_topology_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        support = unpack_mask(mask_records[subject_id]['sampling_support'])
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        crop_mask = unpack_mask(mask_records[subject_id]['crop_mask'])
        region_map = load_region_map(subject_id)
        labels, relation_by_label = topology_candidate_labels(support, region_map)
        for relation in [2, 1, 3]:
            relation_labels = [
                label for label in labels if relation_by_label[label] == relation
            ]
            if not relation_labels:
                continue
            candidate_mask = np.isin(region_map, relation_labels) & crop_mask
            center, slices = first_zero_support_center(
                exclusion, candidate_mask, rng
            )
            if center is not None:
                return subject_id, center, slices, RELATION_NAME[relation]
    raise RuntimeError('Could not construct a support-free topology negative.')


def choose_brain_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        crop_mask = unpack_mask(mask_records[subject_id]['crop_mask'])
        center, slices = first_zero_support_center(exclusion, crop_mask, rng)
        if center is not None:
            return subject_id, center, slices
    raise RuntimeError('Could not construct a support-free brain negative.')


def balanced_positive_schedule(fold_train_df, rng):
    schedule = []
    for lesion_bin in ['tiny', 'small', 'medium', 'large', 'very_large']:
        pool_df = fold_train_df[
            (fold_train_df['trial8_effective_lesion_volume_bin'] == lesion_bin)
            & fold_train_df['trial8_sampling_positive'].map(as_bool)
        ].copy().sort_values('subject_id')
        pool = pool_df['subject_id'].astype(str).tolist()
        if not pool:
            raise RuntimeError(f'Positive bin {lesion_bin} is empty in a fold.')
        target_count = max(MINIMUM_POSITIVE_PATCHES_PER_BIN, len(pool))
        selected = pool.copy()
        hard_pool = [
            subject_id for subject_id in pool
            if str(target_by_subject[subject_id]['supervision_mode'])
            == 'hard_and_soft'
        ]
        rng.shuffle(hard_pool)
        if not hard_pool and len(selected) < target_count:
            raise RuntimeError(f'No hard-target repeat pool for {lesion_bin}.')
        repeat_index = 0
        while len(selected) < target_count:
            selected.append(hard_pool[repeat_index % len(hard_pool)])
            repeat_index += 1
        rng.shuffle(selected)
        schedule.extend(selected)
    rng.shuffle(schedule)
    return schedule


sampling_rows = []
schedule_summary_rows = []
for heldout_fold in range(NUMBER_OF_FOLDS):
    rng = np.random.default_rng(SEED + heldout_fold * 100_003)
    fold_train_df = train_df[train_df['development_fold'] != heldout_fold].copy()
    fold_heldout_df = train_df[train_df['development_fold'] == heldout_fold].copy()
    positive_schedule = balanced_positive_schedule(fold_train_df, rng)
    positive_count = len(positive_schedule)
    topology_negative_count = int(math.ceil(positive_count / 4.0))
    brain_negative_count = int(math.floor(positive_count / 4.0))

    topology_pool = fold_train_df[
        fold_train_df['trial8_atlas_enabled'].map(as_bool)
        & fold_train_df['trial8_sampling_positive'].map(as_bool)
        & (fold_train_df['trial8_supervision_mode'] == 'hard_and_soft')
    ]['subject_id'].astype(str).tolist()
    brain_pool = fold_train_df[
        fold_train_df['trial8_supervision_mode'] != 'soft_only_subvoxel'
    ]['subject_id'].astype(str).tolist()

    for sample_index, subject_id in enumerate(positive_schedule):
        center, patch_support_voxels = choose_positive_center(
            subject_id, rng
        )
        metadata = target_by_subject[subject_id]
        sampling_rows.append({
            'heldout_fold': heldout_fold,
            'sample_index': sample_index,
            'sample_type': 'positive',
            'subject_id': subject_id,
            'subject_development_fold': int(metadata['development_fold']),
            'lesion_volume_bin': metadata['effective_lesion_volume_bin'],
            'supervision_mode': metadata['supervision_mode'],
            'center_x': int(center[0]),
            'center_y': int(center[1]),
            'center_z': int(center[2]),
            'patch_sampling_support_voxels': patch_support_voxels,
            'patch_exclusion_support_voxels': patch_support_voxels,
            'topology_relation': '',
        })

    for local_index in range(topology_negative_count):
        subject_id, center, slices, relation = choose_topology_negative(
            int(rng.integers(0, len(topology_pool))), topology_pool, rng
        )
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        sampling_rows.append({
            'heldout_fold': heldout_fold,
            'sample_index': positive_count + local_index,
            'sample_type': 'topology_negative',
            'subject_id': subject_id,
            'subject_development_fold': int(
                target_by_subject[subject_id]['development_fold']
            ),
            'lesion_volume_bin': target_by_subject[subject_id][
                'effective_lesion_volume_bin'
            ],
            'supervision_mode': target_by_subject[subject_id]['supervision_mode'],
            'center_x': int(center[0]),
            'center_y': int(center[1]),
            'center_z': int(center[2]),
            'patch_sampling_support_voxels': 0,
            'patch_exclusion_support_voxels': int(exclusion[slices].sum()),
            'topology_relation': relation,
        })

    for local_index in range(brain_negative_count):
        subject_id, center, slices = choose_brain_negative(
            int(rng.integers(0, len(brain_pool))), brain_pool, rng
        )
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        sampling_rows.append({
            'heldout_fold': heldout_fold,
            'sample_index': positive_count + topology_negative_count + local_index,
            'sample_type': 'brain_negative',
            'subject_id': subject_id,
            'subject_development_fold': int(
                target_by_subject[subject_id]['development_fold']
            ),
            'lesion_volume_bin': target_by_subject[subject_id][
                'effective_lesion_volume_bin'
            ],
            'supervision_mode': target_by_subject[subject_id]['supervision_mode'],
            'center_x': int(center[0]),
            'center_y': int(center[1]),
            'center_z': int(center[2]),
            'patch_sampling_support_voxels': 0,
            'patch_exclusion_support_voxels': int(exclusion[slices].sum()),
            'topology_relation': '',
        })

    fold_rows = [row for row in sampling_rows if row['heldout_fold'] == heldout_fold]
    fold_schedule_df = pd.DataFrame(fold_rows)
    positive_subjects = set(fold_schedule_df.loc[
        fold_schedule_df['sample_type'] == 'positive', 'subject_id'
    ].astype(str))
    expected_positive_subjects = set(fold_train_df.loc[
        fold_train_df['trial8_sampling_positive'].map(as_bool), 'subject_id'
    ].astype(str))
    soft_only_count = int((
        (fold_schedule_df['sample_type'] == 'positive')
        & (fold_schedule_df['supervision_mode'] == 'soft_only_subvoxel')
    ).sum())
    schedule_summary_rows.append({
        'heldout_fold': heldout_fold,
        'fit_subjects': len(fold_train_df),
        'heldout_subjects': len(fold_heldout_df),
        'positive_patches': positive_count,
        'topology_negative_patches': topology_negative_count,
        'brain_negative_patches': brain_negative_count,
        'total_patches': len(fold_schedule_df),
        'positive_fraction': float(positive_count / len(fold_schedule_df)),
        'all_positive_subjects_covered': (
            expected_positive_subjects.issubset(positive_subjects)
        ),
        'heldout_subjects_sampled': int(
            fold_schedule_df['subject_id'].isin(
                fold_heldout_df['subject_id']
            ).sum()
        ),
        'soft_only_positive_patches': soft_only_count,
        'soft_only_case_in_training_partition': bool(
            (fold_train_df['trial8_supervision_mode'] == 'soft_only_subvoxel').any()
        ),
    })

sampling_audit_df = pd.DataFrame(sampling_rows)
schedule_summary_df = pd.DataFrame(schedule_summary_rows)
atomic_write_csv(sampling_audit_df, SAMPLING_AUDIT_PATH)
atomic_write_csv(schedule_summary_df, FOLD_SCHEDULE_SUMMARY_PATH)






soft_only_subject = target_audit_df.loc[
    target_audit_df['supervision_mode'] == 'soft_only_subvoxel', 'subject_id'
].iloc[0]
soft_only_components = []
for epoch in [0, 1]:
    rng = np.random.default_rng(SEED + epoch * 1_000_003)
    support = unpack_mask(mask_records[soft_only_subject]['sampling_support'])
    labels, count = ndimage_label_26n(support)
    center, patch_voxels = choose_positive_center(
        soft_only_subject, rng, component_index=epoch
    )
    selected_component = int(labels[tuple(np.clip(
        center, 0, np.asarray(labels.shape) - 1
    ))])
    
    
    soft_only_components.append({
        'epoch': epoch,
        'available_components': count,
        'intended_component': 1 + (epoch % count),
        'center_component_after_jitter': selected_component,
        'patch_support_voxels': patch_voxels,
    })


def soft_dice_per_case(logits, soft_target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = soft_target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def soft_focal_bce_per_case(logits, soft_target, gamma=2.0):
    logits = logits.float()
    target = soft_target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, raw.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def focal_tversky_per_case(
    logits, binary_target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = binary_target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error_floor = 1e-6
    return (
        (1.0 - tversky).clamp_min(error_floor).pow(gamma)
        - float(error_floor ** gamma)
    )


def revised_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    dice = soft_dice_per_case(logits, soft_target)
    focal_bce = soft_focal_bce_per_case(logits, soft_target)
    hard_tversky = focal_tversky_per_case(logits, binary_target)
    hard_contribution = 0.20 * hard_tversky * hard_weight.reshape(-1)
    total_per_case = 0.50 * dice + 0.30 * focal_bce + hard_contribution
    return total_per_case.mean(), {
        'soft_dice': dice.mean(),
        'soft_focal_bce': focal_bce.mean(),
        'hard_focal_tversky': hard_tversky.mean(),
        'hard_contribution': hard_contribution.mean(),
    }


def load_target_patch(subject_id, positive):
    row = train_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask']) > 0
        hard_weight = scalar(data, 'binary_loss_weight')
    if positive:
        support = binary > 0
        if not support.any():
            support = soft > 0
        center = np.rint(np.argwhere(support).mean(axis=0)).astype(int)
    else:
        coordinates = np.argwhere(crop_mask)
        center = coordinates[len(coordinates) // 2]
    slices = patch_slices_from_center(binary.shape, center, PATCH_SIZE)
    return (
        extract_patch(binary, slices, PATCH_SIZE),
        extract_patch(soft, slices, PATCH_SIZE),
        hard_weight,
    )


standard_subject = target_audit_df[
    target_audit_df['supervision_mode'] == 'hard_and_soft'
].sort_values('effective_target_volume_ml').iloc[len(
    target_audit_df[target_audit_df['supervision_mode'] == 'hard_and_soft']
) // 2]['subject_id']
true_empty_subject = target_audit_df.loc[
    target_audit_df['supervision_mode'] == 'true_empty', 'subject_id'
].iloc[0]

loss_probe_cases = [
    ('hard_and_soft', str(standard_subject), True),
    ('soft_only_subvoxel', str(soft_only_subject), True),
    ('true_empty', str(true_empty_subject), False),
]
loss_rows = []
for mode, subject_id, positive in loss_probe_cases:
    binary_patch, soft_patch, hard_weight = load_target_patch(
        subject_id, positive
    )
    logits = torch.full(
        (1, 1, *PATCH_SIZE), -2.0, dtype=torch.float32, requires_grad=True
    )
    soft_tensor = torch.from_numpy(soft_patch[None, None]).float()
    binary_tensor = torch.from_numpy(binary_patch[None, None]).float()
    hard_tensor = torch.as_tensor([hard_weight], dtype=torch.float32)
    loss, parts = revised_segmentation_loss(
        logits, soft_tensor, binary_tensor, hard_tensor
    )
    loss.backward()
    gradient = logits.grad.detach()
    support = soft_tensor > 0
    support_gradient = gradient[support]
    background_gradient = gradient[~support]
    loss_rows.append({
        'supervision_mode': mode,
        'subject_id': subject_id,
        'binary_voxels': int(binary_patch.sum()),
        'soft_sum': float(soft_patch.sum(dtype=np.float64)),
        'hard_loss_weight': hard_weight,
        'total_loss': float(loss.detach()),
        'soft_dice_loss': float(parts['soft_dice'].detach()),
        'soft_focal_bce': float(parts['soft_focal_bce'].detach()),
        'hard_focal_tversky': float(parts['hard_focal_tversky'].detach()),
        'hard_contribution': float(parts['hard_contribution'].detach()),
        'gradient_finite': bool(torch.isfinite(gradient).all()),
        'gradient_norm': float(torch.linalg.vector_norm(gradient)),
        'support_voxels_with_nonzero_gradient': int(
            (support_gradient.abs() > 0).sum()
        ),
        'support_voxels': int(support.sum()),
        'mean_absolute_support_gradient': (
            float(support_gradient.abs().mean()) if support.any() else np.nan
        ),
        'mean_absolute_background_gradient': float(
            background_gradient.abs().mean()
        ),
    })

loss_audit_df = pd.DataFrame(loss_rows)
atomic_write_csv(loss_audit_df, LOSS_AUDIT_PATH)






scaler_checks = {
    'five_fold_scalers_written': len(scaler_audit_df) == NUMBER_OF_FOLDS,
    'fold_scaler_fit_counts_match': bool(
        scaler_audit_df['fit_subjects'].eq(
            scaler_audit_df['expected_fit_subjects']
        ).all()
    ),
    'fold_scaler_fit_and_heldout_disjoint': bool(
        scaler_audit_df['fit_and_heldout_disjoint'].all()
    ),
    'heldout_sidecars_not_read_for_own_scaler': not bool(
        scaler_audit_df['heldout_sidecars_read_for_scaler'].any()
    ),
    'targets_not_read_for_scalers': not bool(
        scaler_audit_df['target_arrays_read_for_scaler'].any()
    ),
    'all_fold_scaler_features_supported': bool(
        scaler_audit_df[[
            'territory_features_supported', 'vessel_features_supported',
            'case_features_supported',
        ]].all().all()
    ),
    'all_fold_scaled_features_nonconstant': bool(
        scaler_audit_df[[
            'territory_scaled_features_nonconstant',
            'vessel_scaled_features_nonconstant',
            'case_scaled_features_nonconstant',
        ]].all().all()
    ),
    'all_fold_scaler_statistics_finite': bool(
        scaler_audit_df['all_statistics_finite'].all()
    ),
    'all_fold_scaler_stds_positive': bool(
        scaler_audit_df['all_scaled_stds_positive'].all()
    ),
}

target_checks = {
    'exactly_102_train_targets_rechecked': len(target_audit_df) == 102,
    'no_locked_validation_or_test_targets_read': True,
    'exactly_one_soft_only_subvoxel_case': int(
        (target_audit_df['supervision_mode'] == 'soft_only_subvoxel').sum()
    ) == 1,
    'exactly_one_true_empty_train_case': int(
        (target_audit_df['supervision_mode'] == 'true_empty').sum()
    ) == 1,
    'soft_only_case_has_five_sampling_support_voxels': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'sampling_support_voxels',
        ].eq(5).all()
    ),
    'soft_only_case_hard_loss_disabled': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'binary_loss_weight',
        ].eq(0.0).all()
    ),
    'soft_only_case_excluded_from_hard_metrics': not bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'hard_metric_eligible',
        ].any()
    ),
    'soft_only_case_has_no_hard_target': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'binary_voxels',
        ].eq(0).all()
    ),
    'soft_only_case_retains_soft_target': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'soft_nonzero_voxels',
        ].eq(5).all()
    ),
    'hard_and_soft_cases_keep_hard_loss': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'hard_and_soft',
            'binary_loss_weight',
        ].eq(1.0).all()
    ),
    'all_sampling_support_inside_crop': bool(
        target_audit_df['support_inside_crop_fraction'].ge(0.999999).all()
    ),
}

positive_df = sampling_audit_df[
    sampling_audit_df['sample_type'] == 'positive'
]
negative_df = sampling_audit_df[
    sampling_audit_df['sample_type'].isin(
        ['topology_negative', 'brain_negative']
    )
]
sampling_checks = {
    'five_fold_schedules_created': len(schedule_summary_df) == NUMBER_OF_FOLDS,
    'no_heldout_subject_sampled': bool(
        schedule_summary_df['heldout_subjects_sampled'].eq(0).all()
    ),
    'every_fold_train_positive_subject_covered': bool(
        schedule_summary_df['all_positive_subjects_covered'].all()
    ),
    'positive_fraction_between_0p60_and_0p70': bool(
        schedule_summary_df['positive_fraction'].between(0.60, 0.70).all()
    ),
    'all_positive_patches_retain_sampling_support': bool(
        positive_df['patch_sampling_support_voxels'].gt(0).all()
    ),
    'all_negative_patches_exclude_hard_and_soft_support': bool(
        negative_df['patch_exclusion_support_voxels'].eq(0).all()
    ),
    'all_topology_negatives_have_relation': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'topology_relation',
        ].astype(str).str.len().gt(0).all()
    ),
    'soft_only_case_never_used_as_negative': not bool(
        negative_df['supervision_mode'].eq('soft_only_subvoxel').any()
    ),
    'soft_only_case_at_most_once_per_training_epoch': bool(
        schedule_summary_df['soft_only_positive_patches'].le(
            MAXIMUM_SOFT_ONLY_PATCHES_PER_EPOCH
        ).all()
    ),
    'soft_only_case_sampled_once_exactly_when_in_training_partition': bool(
        schedule_summary_df['soft_only_positive_patches'].eq(
            schedule_summary_df[
                'soft_only_case_in_training_partition'
            ].astype(int)
        ).all()
    ),
    'every_positive_volume_bin_has_at_least_twenty_patches_per_fold': bool(
        positive_df.groupby(
            ['heldout_fold', 'lesion_volume_bin']
        ).size().reindex(
            pd.MultiIndex.from_product([
                range(NUMBER_OF_FOLDS),
                ['tiny', 'small', 'medium', 'large', 'very_large'],
            ]),
            fill_value=0,
        ).ge(MINIMUM_POSITIVE_PATCHES_PER_BIN).all()
    ),
    'soft_only_two_components_alternate_across_two_epochs': (
        len(soft_only_components) == 2
        and {record['intended_component'] for record in soft_only_components}
        == {1, 2}
        and all(record['patch_support_voxels'] > 0 for record in soft_only_components)
    ),
}

soft_loss_row = loss_audit_df[
    loss_audit_df['supervision_mode'] == 'soft_only_subvoxel'
].iloc[0]
loss_checks = {
    'three_supervision_modes_smoke_tested': set(
        loss_audit_df['supervision_mode']
    ) == {'hard_and_soft', 'soft_only_subvoxel', 'true_empty'},
    'all_probe_losses_finite': bool(
        np.isfinite(loss_audit_df[[
            'total_loss', 'soft_dice_loss', 'soft_focal_bce',
            'hard_focal_tversky', 'hard_contribution',
        ]].to_numpy(dtype=np.float64)).all()
    ),
    'all_probe_gradients_finite_and_nonzero': bool(
        loss_audit_df['gradient_finite'].all()
        and loss_audit_df['gradient_norm'].gt(0).all()
    ),
    'soft_only_hard_contribution_exactly_zero': math.isclose(
        float(soft_loss_row['hard_contribution']), 0.0, abs_tol=1e-12
    ),
    'soft_only_all_support_voxels_receive_gradient': (
        int(soft_loss_row['support_voxels_with_nonzero_gradient'])
        == int(soft_loss_row['support_voxels'])
        and int(soft_loss_row['support_voxels']) == 5
    ),
    'soft_only_support_gradient_exceeds_background_gradient': (
        float(soft_loss_row['mean_absolute_support_gradient'])
        > float(soft_loss_row['mean_absolute_background_gradient'])
    ),
}

all_checks = {
    **upstream_checks,
    **fold_cross_checks,
    **manifest_checks,
    **scaler_checks,
    **target_checks,
    **sampling_checks,
    **loss_checks,
    'architecture_parameter_count_frozen': True,
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
    'training_not_performed': True,
    'study_trainable_parameters_are_zero': True,
}
preflight_passed = bool(all(all_checks.values()))

training_policy = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'policy_version': PREFLIGHT_VERSION,
    'scientific_core': (
        'unchanged 15-channel residual 3D U-Net plus heterogeneous '
        'relation-aware GATv2 and bounded zero-init multiscale FiLM'
    ),
    'scientific_core_changed': False,
    'total_parameters': EXPECTED_PARAMETERS,
    'development_design': 'five_fold_cross_validation_with_locked_confirmation',
    'fold_count': NUMBER_OF_FOLDS,
    'fold_scalers': fold_scaler_paths,
    'fold_scaler_sha256': fold_scaler_hashes,
    'patch_size': list(PATCH_SIZE),
    'input_channels': 15,
    'positive_sampling': (
        'all development-train positive subjects once; each lesion-volume '
        'bin padded to at least 20 using hard-target cases only'
    ),
    'minimum_positive_patches_per_bin': MINIMUM_POSITIVE_PATCHES_PER_BIN,
    'soft_only_subvoxel_policy': {
        'positive_center_mask': 'target_soft > 0',
        'maximum_samples_per_epoch': MAXIMUM_SOFT_ONLY_PATCHES_PER_EPOCH,
        'component_selection': 'alternate connected components by epoch',
        'hard_loss_weight': 0.0,
        'hard_metric_eligible': False,
        'negative_sampling_eligible': False,
    },
    'negative_sampling': (
        'topology and brain negatives require zero voxels from '
        'target_binary OR target_soft>0 support'
    ),
    'epoch_resampling': (
        'rebuild deterministically from fold and epoch seeds; rotate repeated '
        'hard-target subjects so padding does not repeatedly favor the same cases'
    ),
    'positive_fraction_target': [0.60, 0.70],
    'volume_case_loss_weighting': False,
    'clinical_window_recalibration': False,
    'frozen_input_channels_used_unchanged': True,
    'loss': {
        'soft_dice_weight': 0.50,
        'soft_focal_bce_weight': 0.30,
        'hard_focal_tversky_weight': 0.20,
        'hard_focal_tversky_alpha_false_positive': 0.30,
        'hard_focal_tversky_beta_false_negative': 0.70,
        'hard_focal_tversky_gamma': 0.75,
        'top_k_ohem': False,
    },
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'soft_only_auxiliary_policy': (
        'main-resolution soft loss only; exclude from half/quarter losses to '
        'avoid erasing five-voxel subvoxel support during downsampling'
    ),
    'territory_node_loss_weight': 0.02,
    'territory_positive_weight': 4.0,
    'territory_node_supervision_policy': (
        'hard-target-eligible and node-valid subjects only; soft-only subvoxel '
        'case excluded because its empty hard mask is not a true negative'
    ),
    'batch_size': 1,
    'gradient_accumulation_steps': 4,
    'effective_batch_size': 4,
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'metrics_printed_every_epoch': True,
    'early_stopping_minimum_epoch': 20,
    'early_stopping_patience_validations': 15,
    'early_stopping_minimum_delta': 0.002,
    'optimizer': 'AdamW',
    'learning_rate': 8e-5,
    'weight_decay': 2e-3,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'ema_decay': 0.99,
    'validation_model': 'exponential_moving_average',
    'intensity_gain_range': [0.92, 1.08],
    'intensity_offset_range': [-0.02, 0.02],
    'noise_probability': 0.50,
    'maximum_noise_standard_deviation': 0.015,
    'gamma_probability': 0.40,
    'gamma_range': [0.90, 1.10],
    'spatial_affine_probability': 0.60,
    'spatial_rotation_degrees': 8.0,
    'spatial_scale_range': [0.92, 1.08],
    'spatial_translation_fraction_xy': 0.05,
    'spatial_translation_fraction_z': 0.025,
    'left_right_reflection': False,
    'post_augmentation_positive_support_required': True,
    'augmentation_retry_then_identity_fallback': True,
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': 0.25,
    'checkpoint_metric': 'heldout_positive_mean_dice',
    'threshold_grid': [
        round(value, 2) for value in np.arange(0.10, 0.81, 0.05)
    ],
    'heldout_fold_metrics': [
        'positive_case_mean_dice', 'global_dice', 'lesion_precision',
        'lesion_recall', 'lesion_f1', 'absolute_volume_error_ml',
        'false_positive_volume_on_true_empty_ml',
    ],
    'validation_inference': 'full_volume_sliding_window_not_sampled_patches',
    'hard_metric_exclusion': ['soft_only_subvoxel'],
    'locked_validation_policy': (
        'do not read during five-fold development; use once only after '
        'training policy and hyperparameters are frozen'
    ),
    'test_policy': 'sealed until final evaluation',
    'fold0_pilot_training_authorized': preflight_passed,
    'full_fivefold_training_authorized': False,
}
atomic_write_json(training_policy, TRAINING_POLICY_PATH)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': all_checks,
    'target_policy_manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'target_policy_manifest_sha256': target_policy_manifest_sha256,
    'fold_manifest_path': str(FOLD_MANIFEST_PATH),
    'fold_manifest_sha256': fold_manifest_sha256,
    'training_policy_path': str(TRAINING_POLICY_PATH),
    'fold_scaler_paths': fold_scaler_paths,
    'fold_scaler_sha256': fold_scaler_hashes,
    'target_audit_path': str(TARGET_AUDIT_PATH),
    'scaler_audit_path': str(SCALER_AUDIT_PATH),
    'sampling_audit_path': str(SAMPLING_AUDIT_PATH),
    'loss_audit_path': str(LOSS_AUDIT_PATH),
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'training_performed': False,
    'study_trainable_parameters': 0,
    'architecture_parameter_count': EXPECTED_PARAMETERS,
    'architecture_changed': False,
    'fold0_pilot_training_authorized': preflight_passed,
    'full_fivefold_training_authorized': False,
    'next_block': 'isolated fold-0 pilot training with per-epoch metrics',
}
atomic_write_json(gate, PREFLIGHT_GATE_PATH)

print('\nFold-specific scaler audit:')
display(scaler_audit_df)
print('\nTrain target-policy recheck:')
display(target_audit_df.groupby(
    ['supervision_mode', 'effective_lesion_volume_bin'], as_index=False
).agg(
    subjects=('subject_id', 'count'),
    sampling_support_voxels=('sampling_support_voxels', 'sum'),
    minimum_effective_volume_ml=('effective_target_volume_ml', 'min'),
    maximum_effective_volume_ml=('effective_target_volume_ml', 'max'),
))
print('\nFive-fold sampling schedules:')
display(schedule_summary_df)
print('\nSoft-only component alternation:')
display(pd.DataFrame(soft_only_components))
print('\nReal-patch loss smoke audit:')
display(loss_audit_df)
print('\nPreflight checks:')
display(pd.DataFrame({
    'check': list(all_checks.keys()),
    'passed': list(all_checks.values()),
}))

if not preflight_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 FIVE-FOLD LOADER/LOSS/SAMPLER PREFLIGHT FAILED. '
        'Do not train. Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 FIVE-FOLD LOADER/LOSS/SAMPLER PREFLIGHT PASSED')
print('=' * 80)
print(f'Development subjects:               {len(train_df)}')
print(f'Fold-specific graph scalers:        {len(scaler_audit_df)}')
print(f'Patch size:                         {PATCH_SIZE}')
print('Volume-based case loss weights:     disabled')
print('Top-k OHEM:                         disabled')
print('Soft-only subvoxel supervision:     enabled')
print('Locked validation targets read:     False')
print('Test targets read:                  False')
print('Study trainable parameters:         0 (preflight only)')
print(f'Architecture parameters:            {EXPECTED_PARAMETERS:,} (unchanged)')
print(f'Fold-0 pilot training authorized:   {preflight_passed}')
print('Full five-fold training authorized: False')
print(f'Training policy:                    {TRAINING_POLICY_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print('\nNext: run the isolated fold-0 pilot and print metrics every epoch.')


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')
scipy = ensure_package('scipy')

import torch.nn.functional as F
from scipy import ndimage
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
MODEL_VERSION = 'trial8_vascufate_net_v1'
TARGET_POLICY_VERSION = 'v3_soft_only_subvoxel_train5fold_balanced'
PREFLIGHT_VERSION = 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_PARAMETERS = 15_052_628
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_GLOBAL_SCALER_SHA256 = (
    '89efb14870de2872f88d0c0650efc0d26e3179da5cbc5751fe1c0c81a374d1c6'
)

NUMBER_OF_FOLDS = 5
SEED = 20260924
PATCH_SIZE = (128, 128, 64)
MINIMUM_POSITIVE_PATCHES_PER_BIN = 20
MAXIMUM_SOFT_ONLY_PATCHES_PER_EPOCH = 1
MAXIMUM_NEGATIVE_CENTER_ATTEMPTS = 512
SCALER_EPSILON = 1e-6

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
TARGET_POLICY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'target_integrity'
    / TARGET_POLICY_VERSION
)
TARGET_POLICY_GATE_PATH = (
    TARGET_POLICY_ROOT / 'target_policy_fold_refreeze_gate.json'
)

GRAPH_VERIFICATION_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
)
GRAPH_SCHEMA_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)
GRAPH_VERIFICATION_GATE_PATH = (
    GRAPH_VERIFICATION_ROOT / 'graph_verification_gate.json'
)

MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_preflight'
    / MODEL_VERSION
    / 'v1_train_only_scaler_ragatv2_film'
)
MODEL_PREFLIGHT_GATE_PATH = (
    MODEL_PREFLIGHT_ROOT / 'model_scaler_preflight_gate.json'
)
ARCHITECTURE_CONFIG_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'
GLOBAL_SCALER_PATH = MODEL_PREFLIGHT_ROOT / 'train_only_graph_scaler.npz'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_ROOT = OUTPUT_ROOT / 'fold_scalers'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SCALER_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_AUDIT_PATH = OUTPUT_ROOT / 'train_target_policy_recheck.csv'
SCALER_AUDIT_PATH = OUTPUT_ROOT / 'fold_scaler_audit.csv'
SAMPLING_AUDIT_PATH = OUTPUT_ROOT / 'fold_sampling_audit.csv'
FOLD_SCHEDULE_SUMMARY_PATH = OUTPUT_ROOT / 'fold_schedule_summary.csv'
LOSS_AUDIT_PATH = OUTPUT_ROOT / 'loss_smoke_audit.csv'
TRAINING_POLICY_PATH = OUTPUT_ROOT / 'fivefold_training_policy.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'fivefold_loader_loss_sampler_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def patch_slices_from_center(shape, center, patch_size):
    slices = []
    for size, coordinate, patch in zip(shape, center, patch_size):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def extract_patch(array, slices, patch_size):
    cropped = np.asarray(array)[slices]
    padding = []
    for observed, expected in zip(cropped.shape, patch_size):
        missing = max(int(expected) - int(observed), 0)
        lower = missing // 2
        padding.append((lower, missing - lower))
    if any(lower or upper for lower, upper in padding):
        cropped = np.pad(cropped, padding, mode='constant')
    output_slices = tuple(
        slice(0, min(cropped.shape[axis], patch_size[axis]))
        for axis in range(3)
    )
    return cropped[output_slices]


def pack_mask(mask):
    mask = np.asarray(mask, dtype=bool)
    return {
        'shape': tuple(int(value) for value in mask.shape),
        'packed': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def unpack_mask(record):
    voxel_count = int(np.prod(record['shape']))
    values = np.unpackbits(
        record['packed'], count=voxel_count, bitorder='little'
    )
    return values.reshape(record['shape']).astype(bool)


def subject_hash(subjects):
    return hashlib.sha256(
        '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    ).hexdigest()



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': PREFLIGHT_VERSION,
        'preflight_passed': False,
        'status': 'running_or_incomplete',
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'training_performed': False,
        'fold0_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
    },
    PREFLIGHT_GATE_PATH,
)






required_paths = [
    TARGET_POLICY_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    TARGET_POLICY_GATE_PATH,
    GRAPH_SCHEMA_PATH,
    GRAPH_VERIFICATION_GATE_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
    ARCHITECTURE_CONFIG_PATH,
    GLOBAL_SCALER_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 five-fold preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

target_policy_manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)
graph_schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
global_scaler_sha256 = file_sha256(GLOBAL_SCALER_PATH)
architecture_config_sha256 = file_sha256(ARCHITECTURE_CONFIG_PATH)

observed_expected_hashes = {
    'target_policy_manifest': (
        target_policy_manifest_sha256,
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256,
    ),
    'fold_manifest': (fold_manifest_sha256, EXPECTED_FOLD_MANIFEST_SHA256),
    'graph_schema': (graph_schema_sha256, EXPECTED_GRAPH_SCHEMA_SHA256),
    'global_scaler_schema_template': (
        global_scaler_sha256,
        EXPECTED_GLOBAL_SCALER_SHA256,
    ),
}
hash_errors = [
    f'{name}: observed {observed}, expected {expected}'
    for name, (observed, expected) in observed_expected_hashes.items()
    if observed != expected
]
if hash_errors:
    raise RuntimeError('Frozen hash mismatch:\n' + '\n'.join(hash_errors))

target_policy_gate = json.loads(
    TARGET_POLICY_GATE_PATH.read_text(encoding='utf-8')
)
graph_gate = json.loads(
    GRAPH_VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
model_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
architecture_config = json.loads(
    ARCHITECTURE_CONFIG_PATH.read_text(encoding='utf-8')
)

upstream_checks = {
    'target_policy_frozen': target_policy_gate.get('policy_frozen') is True,
    'target_policy_test_targets_sealed': (
        target_policy_gate.get('test_target_arrays_read') is False
    ),
    'target_policy_npz_files_unchanged': (
        target_policy_gate.get('frozen_npz_files_modified') is False
    ),
    'target_policy_manifest_hash_matches_gate': (
        target_policy_gate.get('target_policy_manifest_sha256')
        == target_policy_manifest_sha256
    ),
    'fold_manifest_hash_matches_gate': (
        target_policy_gate.get('fold_manifest_sha256') == fold_manifest_sha256
    ),
    'graph_verification_passed': graph_gate.get('verification_passed') is True,
    'graph_verification_target_free': graph_gate.get('target_arrays_read') is False,
    'model_preflight_passed': model_gate.get('preflight_passed') is True,
    'model_preflight_architecture_hash_matches': (
        model_gate.get('architecture_config_sha256')
        == architecture_config_sha256
    ),
    'model_preflight_scaler_hash_matches': (
        model_gate.get('scaler_sha256') == global_scaler_sha256
    ),
    'architecture_parameter_count_unchanged': (
        int(model_gate.get('total_parameters', -1)) == EXPECTED_PARAMETERS
    ),
    'architecture_config_parameter_count_unchanged': (
        int(architecture_config.get('total_parameters', -1))
        == EXPECTED_PARAMETERS
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream gates failed: ' + ', '.join(failed))

manifest_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id', 'dev_split', 'vascufate_v2_npz_path', 'trial8_npz_sha256',
    'trial8_atlas_enabled', 'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path', 'trial8_graph_sidecar_sha256',
    'trial8_supervision_mode', 'trial8_effective_target_volume_ml',
    'trial8_effective_lesion_volume_bin', 'trial8_sampling_positive',
    'trial8_negative_sampling_exclusion_required',
    'trial8_hard_metric_eligible', 'trial8_target_policy_frozen',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        f'Target-policy manifest is missing columns: {missing_manifest_columns}'
    )
required_fold_columns = {
    'subject_id', 'frozen_dev_split', 'development_fold',
    'development_role', 'train_only_supervision_mode',
    'train_only_effective_lesion_volume_bin',
    'target_policy_manifest_sha256',
}
missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
if missing_fold_columns:
    raise RuntimeError(f'Fold manifest is missing columns: {missing_fold_columns}')

manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
    raise RuntimeError('Target-policy and fold manifests have different subjects.')
fold_cross_checks = {
    'fold_splits_match_policy_manifest': fold_df['frozen_dev_split'].astype(
        str
    ).equals(manifest_df['dev_split'].astype(str)),
    'fold_policy_hash_matches_for_all_rows': fold_df[
        'target_policy_manifest_sha256'
    ].astype(str).eq(target_policy_manifest_sha256).all(),
    'train_fold_supervision_matches_policy': fold_df.loc[
        fold_df['frozen_dev_split'] == 'train',
        'train_only_supervision_mode',
    ].astype(str).reset_index(drop=True).equals(
        manifest_df.loc[
            manifest_df['dev_split'] == 'train',
            'trial8_supervision_mode',
        ].astype(str).reset_index(drop=True)
    ),
    'train_fold_volume_bins_match_policy': fold_df.loc[
        fold_df['frozen_dev_split'] == 'train',
        'train_only_effective_lesion_volume_bin',
    ].astype(str).reset_index(drop=True).equals(
        manifest_df.loc[
            manifest_df['dev_split'] == 'train',
            'trial8_effective_lesion_volume_bin',
        ].astype(str).reset_index(drop=True)
    ),
}
if not all(fold_cross_checks.values()):
    failed = [name for name, passed in fold_cross_checks.items() if not passed]
    raise RuntimeError('Fold-manifest cross-checks failed: ' + ', '.join(failed))
manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
manifest_df['development_role'] = fold_df['development_role'].astype(str)

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_are_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
    'policy_frozen_for_all': manifest_df['trial8_target_policy_frozen'].map(
        as_bool
    ).all(),
    'exactly_102_train_fold_assignments': (
        manifest_df.loc[manifest_df['dev_split'] == 'train', 'development_fold']
        .between(0, 4).all()
    ),
    'locked_validation_and_test_have_no_fold': manifest_df.loc[
        manifest_df['dev_split'].isin(['val', 'test']), 'development_fold'
    ].eq(-1).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Manifest checks failed: ' + ', '.join(failed))

train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != 102:
    raise RuntimeError('Expected exactly 102 train subjects.')

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    territory_edge_index = np.asarray(schema['territory_edge_index'], dtype=np.int64)
    territory_edge_type = np.asarray(schema['territory_edge_type'], dtype=np.int64)

with np.load(GLOBAL_SCALER_PATH, allow_pickle=False) as scaler_template:
    scaler_schema = {
        name: np.asarray(scaler_template[name])
        for name in [
            'territory_feature_names', 'vessel_feature_names',
            'vessel_source_indices', 'case_feature_names',
            'case_source_indices', 'case_scale_mask', 'clip_value',
        ]
    }

print('Trial8 five-fold preflight upstream gates passed.')
print(f'Policy subjects:                    {len(manifest_df)}')
print(f'Split counts:                       {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Target-policy manifest SHA-256:     {target_policy_manifest_sha256}')
print(f'Five-fold manifest SHA-256:         {fold_manifest_sha256}')
print(f'Architecture parameters:            {EXPECTED_PARAMETERS:,} (unchanged)')
print('Locked validation targets read:     False')
print('Test targets read:                  False')
print('Study trainable parameters:         0 (preflight only)')






territory_feature_names = [str(value) for value in scaler_schema[
    'territory_feature_names'
].tolist()]
vessel_feature_names = [str(value) for value in scaler_schema[
    'vessel_feature_names'
].tolist()]
case_feature_names = [str(value) for value in scaler_schema[
    'case_feature_names'
].tolist()]
vessel_source_indices = scaler_schema['vessel_source_indices'].astype(int)
case_source_indices = scaler_schema['case_source_indices'].astype(int)
case_scale_mask = scaler_schema['case_scale_mask'].astype(bool)
scaler_clip = float(np.asarray(scaler_schema['clip_value']).reshape(-1)[0])


def fit_feature_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    constants = np.zeros(len(feature_names), dtype=np.uint8)
    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid fold-scaler support for {feature_names[index]}.'
            )
        counts[index] = int(array.size)
        raw_std = float(array.std())
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
    return means, stds, counts, constants


scaler_rows = []
fold_scaler_paths = {}
fold_scaler_hashes = {}
for heldout_fold in range(NUMBER_OF_FOLDS):
    fit_df = train_df[train_df['development_fold'] != heldout_fold].copy()
    heldout_df = train_df[train_df['development_fold'] == heldout_fold].copy()
    expected_fit_count = 102 - len(heldout_df)

    territory_accumulators = [[] for _ in territory_feature_names]
    vessel_accumulators = [[] for _ in vessel_feature_names]
    case_accumulators = [[] for _ in case_feature_names]
    fit_subjects = []

    for row in fit_df.sort_values('subject_id').to_dict('records'):
        subject_id = str(row['subject_id'])
        sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
        if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
            raise RuntimeError(f'{subject_id}: graph-sidecar hash changed.')
        with np.load(sidecar_path, allow_pickle=False) as data:
            forbidden = {
                'target_binary', 'target_soft', 'binary_loss_weight',
                'source_target_volume_ml', 'target_binary_volume_ml',
                'target_soft_volume_ml',
            }
            if forbidden & set(data.files):
                raise RuntimeError(f'{subject_id}: target found in graph sidecar.')
            node_features = np.asarray(data['node_features'], dtype=np.float32)
            node_feature_valid = np.asarray(data['node_feature_valid']) > 0
            node_valid = np.asarray(data['node_valid']) > 0
            vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
            vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
            vessel_presence = np.asarray(data['vessel_presence']) > 0
            case_features = np.asarray(data['case_features'], dtype=np.float32)

        expected_node_shape = (30, len(territory_feature_names))
        if node_features.shape != expected_node_shape:
            raise RuntimeError(
                f'{subject_id}: node feature shape {node_features.shape} != '
                f'{expected_node_shape}.'
            )
        if node_feature_valid.shape != node_features.shape:
            raise RuntimeError(f'{subject_id}: node feature-valid shape differs.')
        if node_valid.shape != (30,):
            raise RuntimeError(f'{subject_id}: node-valid shape differs.')
        if vessel_features.ndim != 2 or vessel_features.shape[0] != 13:
            raise RuntimeError(f'{subject_id}: vessel feature shape differs.')
        if vessel_feature_valid.shape != vessel_features.shape:
            raise RuntimeError(f'{subject_id}: vessel feature-valid shape differs.')
        if vessel_presence.shape != (13,):
            raise RuntimeError(f'{subject_id}: vessel-presence shape differs.')
        if vessel_source_indices.max(initial=-1) >= vessel_features.shape[1]:
            raise RuntimeError(f'{subject_id}: vessel source index is invalid.')
        if case_source_indices.max(initial=-1) >= case_features.size:
            raise RuntimeError(f'{subject_id}: case source index is invalid.')
        if not all(
            np.isfinite(array).all()
            for array in [node_features, vessel_features, case_features]
        ):
            raise RuntimeError(f'{subject_id}: non-finite graph feature found.')

        for feature_index in range(len(territory_feature_names)):
            valid = node_valid & node_feature_valid[:, feature_index]
            territory_accumulators[feature_index].extend(
                node_features[valid, feature_index].astype(float).tolist()
            )
        for model_index, source_index in enumerate(vessel_source_indices):
            valid = vessel_presence & vessel_feature_valid[:, source_index]
            vessel_accumulators[model_index].extend(
                vessel_features[valid, source_index].astype(float).tolist()
            )
        for model_index, source_index in enumerate(case_source_indices):
            case_accumulators[model_index].append(
                float(case_features[source_index])
            )
        fit_subjects.append(subject_id)

    territory_mean, territory_std, territory_count, territory_constant = (
        fit_feature_scaler(territory_accumulators, territory_feature_names)
    )
    vessel_mean, vessel_std, vessel_count, vessel_constant = fit_feature_scaler(
        vessel_accumulators, vessel_feature_names
    )
    case_mean, case_std, case_count, case_constant = fit_feature_scaler(
        case_accumulators, case_feature_names, scale_mask=case_scale_mask
    )

    scaler_path = SCALER_ROOT / f'fold_{heldout_fold}_graph_scaler.npz'
    fit_hash = subject_hash(fit_subjects)
    heldout_subjects = heldout_df['subject_id'].astype(str).tolist()
    atomic_write_npz(
        scaler_path,
        territory_feature_names=np.asarray(territory_feature_names),
        territory_mean=territory_mean,
        territory_std=territory_std,
        territory_count=territory_count,
        vessel_feature_names=np.asarray(vessel_feature_names),
        vessel_source_indices=vessel_source_indices.astype(np.int16),
        vessel_mean=vessel_mean,
        vessel_std=vessel_std,
        vessel_count=vessel_count,
        case_feature_names=np.asarray(case_feature_names),
        case_source_indices=case_source_indices.astype(np.int16),
        case_scale_mask=case_scale_mask.astype(np.uint8),
        case_mean=case_mean,
        case_std=case_std,
        case_count=case_count,
        clip_value=np.asarray(scaler_clip, dtype=np.float32),
        heldout_fold=np.asarray(heldout_fold, dtype=np.int8),
        fit_subject_count=np.asarray(len(fit_subjects), dtype=np.int16),
        fit_subject_hash=np.asarray(fit_hash),
        heldout_subject_hash=np.asarray(subject_hash(heldout_subjects)),
        target_arrays_read=np.asarray(False, dtype=np.uint8),
    )
    scaler_sha256 = file_sha256(scaler_path)
    fold_scaler_paths[heldout_fold] = str(scaler_path)
    fold_scaler_hashes[heldout_fold] = scaler_sha256
    scaler_rows.append({
        'heldout_fold': heldout_fold,
        'fit_subjects': len(fit_subjects),
        'expected_fit_subjects': expected_fit_count,
        'heldout_subjects': len(heldout_subjects),
        'fit_and_heldout_disjoint': not bool(
            set(fit_subjects) & set(heldout_subjects)
        ),
        'fit_subject_hash': fit_hash,
        'heldout_subject_hash': subject_hash(heldout_subjects),
        'territory_features_supported': bool((territory_count > 0).all()),
        'vessel_features_supported': bool((vessel_count > 0).all()),
        'case_features_supported': bool((case_count > 0).all()),
        'territory_scaled_features_nonconstant': not bool(
            territory_constant.any()
        ),
        'vessel_scaled_features_nonconstant': not bool(vessel_constant.any()),
        'case_scaled_features_nonconstant': not bool(
            case_constant[case_scale_mask].any()
        ),
        'all_statistics_finite': all(
            np.isfinite(array).all()
            for array in [
                territory_mean, territory_std, vessel_mean, vessel_std,
                case_mean, case_std,
            ]
        ),
        'all_scaled_stds_positive': bool(
            (territory_std > 0).all()
            and (vessel_std > 0).all()
            and (case_std[case_scale_mask] > 0).all()
        ),
        'heldout_sidecars_read_for_scaler': False,
        'target_arrays_read_for_scaler': False,
        'scaler_path': str(scaler_path),
        'scaler_sha256': scaler_sha256,
    })

scaler_audit_df = pd.DataFrame(scaler_rows)
atomic_write_csv(scaler_audit_df, SCALER_AUDIT_PATH)






target_rows = []
mask_records = {}
train_by_subject = {
    str(row['subject_id']): row for row in train_df.to_dict('records')
}

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'target_binary', 'target_soft', 'binary_loss_weight', 'crop_mask',
            'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: missing target arrays: {missing}')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        binary_loss_weight = scalar(data, 'binary_loss_weight')
        source_volume_ml = scalar(data, 'source_target_volume_ml')
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if not np.isfinite(soft).all():
        raise RuntimeError(f'{subject_id}: soft target is non-finite.')

    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        sampling_support = binary
        expected_hard_weight = 1.0
    elif supervision_mode == 'soft_only_subvoxel':
        sampling_support = soft > 0
        expected_hard_weight = 0.0
    elif supervision_mode == 'true_empty':
        sampling_support = np.zeros_like(binary)
        expected_hard_weight = 1.0
    else:
        raise RuntimeError(f'{subject_id}: invalid train supervision mode.')

    exclusion_support = binary | (soft > 0)
    support_voxels = int(sampling_support.sum())
    positive_expected = as_bool(row['trial8_sampling_positive'])
    if positive_expected != (support_voxels > 0):
        raise RuntimeError(f'{subject_id}: positive-sampling policy mismatch.')
    if not math.isclose(binary_loss_weight, expected_hard_weight):
        raise RuntimeError(f'{subject_id}: hard-loss policy mismatch.')

    mask_records[subject_id] = {
        'sampling_support': pack_mask(sampling_support),
        'exclusion_support': pack_mask(exclusion_support),
        'crop_mask': pack_mask(crop_mask),
    }
    target_rows.append({
        'subject_id': subject_id,
        'development_fold': int(row['development_fold']),
        'supervision_mode': supervision_mode,
        'effective_lesion_volume_bin': str(
            row['trial8_effective_lesion_volume_bin']
        ),
        'effective_target_volume_ml': float(
            row['trial8_effective_target_volume_ml']
        ),
        'sampling_positive': positive_expected,
        'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
        'binary_loss_weight': binary_loss_weight,
        'source_target_volume_ml': source_volume_ml,
        'binary_voxels': int(binary.sum()),
        'soft_nonzero_voxels': int((soft > 0).sum()),
        'soft_sum': float(soft.sum(dtype=np.float64)),
        'sampling_support_voxels': support_voxels,
        'exclusion_support_voxels': int(exclusion_support.sum()),
        'support_inside_crop_fraction': (
            1.0
            if support_voxels == 0
            else float((sampling_support & crop_mask).sum() / support_voxels)
        ),
    })

target_audit_df = pd.DataFrame(target_rows)
atomic_write_csv(target_audit_df, TARGET_AUDIT_PATH)
target_by_subject = target_audit_df.set_index('subject_id').to_dict('index')






region_map_cache = {}


def load_region_map(subject_id):
    if subject_id in region_map_cache:
        return region_map_cache[subject_id]
    row = train_by_subject[subject_id]
    shape = mask_records[subject_id]['sampling_support']['shape']
    if not as_bool(row['trial8_atlas_enabled']):
        region = np.zeros(shape, dtype=np.uint8)
    else:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        image = nib.load(str(atlas_path))
        region = np.asanyarray(image.dataobj)
        if region.ndim == 4 and region.shape[-1] == 1:
            region = region[..., 0]
        region = np.rint(region).astype(np.uint8)
        if region.shape != shape:
            raise RuntimeError(f'{subject_id}: atlas/target shape mismatch.')
        if region.min() < 0 or region.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    region_map_cache[subject_id] = region
    return region


def topology_candidate_labels(support, region_map):
    lesion_labels = set(
        int(value)
        for value in np.unique(region_map[support])
        if 1 <= int(value) <= 30
    )
    if not lesion_labels:
        return [], {}
    relation_priority = {2: 0, 1: 1, 3: 2}
    relation_by_label = {}
    for edge_index in range(territory_edge_index.shape[1]):
        source = int(territory_edge_index[0, edge_index]) + 1
        destination = int(territory_edge_index[1, edge_index]) + 1
        relation = int(territory_edge_type[edge_index])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in relation_priority
        ):
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    ordered = sorted(
        relation_by_label,
        key=lambda label: (relation_priority[relation_by_label[label]], label),
    )
    return ordered, relation_by_label


RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def choose_positive_center(subject_id, rng, component_index=None):
    support = unpack_mask(mask_records[subject_id]['sampling_support'])
    if not support.any():
        raise RuntimeError(f'{subject_id}: positive case has no support.')
    if component_index is None:
        coordinates = np.argwhere(support)
    else:
        labels, count = ndimage_label_26n(support)
        if count < 1:
            raise RuntimeError(f'{subject_id}: support has no component.')
        selected_label = 1 + (int(component_index) % count)
        coordinates = np.argwhere(labels == selected_label)
    center = coordinates[int(rng.integers(0, len(coordinates)))].astype(np.int64)
    jitter = np.asarray([
        int(rng.integers(-12, 13)),
        int(rng.integers(-12, 13)),
        int(rng.integers(-4, 5)),
    ])
    center = np.clip(center + jitter, 0, np.asarray(support.shape) - 1)
    slices = patch_slices_from_center(support.shape, center, PATCH_SIZE)
    patch_support_voxels = int(support[slices].sum())
    return center, patch_support_voxels


def ndimage_label_26n(mask):
    return ndimage.label(mask, structure=np.ones((3, 3, 3), dtype=np.uint8))


def shuffled_coordinates(mask, rng, maximum=MAXIMUM_NEGATIVE_CENTER_ATTEMPTS):
    flat = np.flatnonzero(mask)
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > maximum:
        flat = rng.choice(flat, size=maximum, replace=False)
    else:
        flat = flat.copy()
        rng.shuffle(flat)
    return np.column_stack(np.unravel_index(flat, mask.shape)).astype(np.int64)


def first_zero_support_center(exclusion_support, candidate_mask, rng):
    coordinates = shuffled_coordinates(candidate_mask, rng)
    for center in coordinates:
        slices = patch_slices_from_center(
            exclusion_support.shape, center, PATCH_SIZE
        )
        if not exclusion_support[slices].any():
            return center, slices
    return None, None


def choose_topology_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        support = unpack_mask(mask_records[subject_id]['sampling_support'])
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        crop_mask = unpack_mask(mask_records[subject_id]['crop_mask'])
        region_map = load_region_map(subject_id)
        labels, relation_by_label = topology_candidate_labels(support, region_map)
        for relation in [2, 1, 3]:
            relation_labels = [
                label for label in labels if relation_by_label[label] == relation
            ]
            if not relation_labels:
                continue
            candidate_mask = np.isin(region_map, relation_labels) & crop_mask
            center, slices = first_zero_support_center(
                exclusion, candidate_mask, rng
            )
            if center is not None:
                return subject_id, center, slices, RELATION_NAME[relation]
    raise RuntimeError('Could not construct a support-free topology negative.')


def choose_brain_negative(start_index, pool, rng):
    for attempt in range(len(pool)):
        subject_id = pool[(start_index + attempt) % len(pool)]
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        crop_mask = unpack_mask(mask_records[subject_id]['crop_mask'])
        center, slices = first_zero_support_center(exclusion, crop_mask, rng)
        if center is not None:
            return subject_id, center, slices
    raise RuntimeError('Could not construct a support-free brain negative.')


def balanced_positive_schedule(fold_train_df, rng):
    schedule = []
    for lesion_bin in ['tiny', 'small', 'medium', 'large', 'very_large']:
        pool_df = fold_train_df[
            (fold_train_df['trial8_effective_lesion_volume_bin'] == lesion_bin)
            & fold_train_df['trial8_sampling_positive'].map(as_bool)
        ].copy().sort_values('subject_id')
        pool = pool_df['subject_id'].astype(str).tolist()
        if not pool:
            raise RuntimeError(f'Positive bin {lesion_bin} is empty in a fold.')
        target_count = max(MINIMUM_POSITIVE_PATCHES_PER_BIN, len(pool))
        selected = pool.copy()
        hard_pool = [
            subject_id for subject_id in pool
            if str(target_by_subject[subject_id]['supervision_mode'])
            == 'hard_and_soft'
        ]
        rng.shuffle(hard_pool)
        if not hard_pool and len(selected) < target_count:
            raise RuntimeError(f'No hard-target repeat pool for {lesion_bin}.')
        repeat_index = 0
        while len(selected) < target_count:
            selected.append(hard_pool[repeat_index % len(hard_pool)])
            repeat_index += 1
        rng.shuffle(selected)
        schedule.extend(selected)
    rng.shuffle(schedule)
    return schedule


sampling_rows = []
schedule_summary_rows = []
for heldout_fold in range(NUMBER_OF_FOLDS):
    rng = np.random.default_rng(SEED + heldout_fold * 100_003)
    fold_train_df = train_df[train_df['development_fold'] != heldout_fold].copy()
    fold_heldout_df = train_df[train_df['development_fold'] == heldout_fold].copy()
    positive_schedule = balanced_positive_schedule(fold_train_df, rng)
    positive_count = len(positive_schedule)
    topology_negative_count = int(math.ceil(positive_count / 4.0))
    brain_negative_count = int(math.floor(positive_count / 4.0))

    topology_pool = fold_train_df[
        fold_train_df['trial8_atlas_enabled'].map(as_bool)
        & fold_train_df['trial8_sampling_positive'].map(as_bool)
        & (fold_train_df['trial8_supervision_mode'] == 'hard_and_soft')
    ]['subject_id'].astype(str).tolist()
    brain_pool = fold_train_df[
        fold_train_df['trial8_supervision_mode'] != 'soft_only_subvoxel'
    ]['subject_id'].astype(str).tolist()

    for sample_index, subject_id in enumerate(positive_schedule):
        center, patch_support_voxels = choose_positive_center(
            subject_id, rng
        )
        metadata = target_by_subject[subject_id]
        sampling_rows.append({
            'heldout_fold': heldout_fold,
            'sample_index': sample_index,
            'sample_type': 'positive',
            'subject_id': subject_id,
            'subject_development_fold': int(metadata['development_fold']),
            'lesion_volume_bin': metadata['effective_lesion_volume_bin'],
            'supervision_mode': metadata['supervision_mode'],
            'center_x': int(center[0]),
            'center_y': int(center[1]),
            'center_z': int(center[2]),
            'patch_sampling_support_voxels': patch_support_voxels,
            'patch_exclusion_support_voxels': patch_support_voxels,
            'topology_relation': '',
        })

    for local_index in range(topology_negative_count):
        subject_id, center, slices, relation = choose_topology_negative(
            int(rng.integers(0, len(topology_pool))), topology_pool, rng
        )
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        sampling_rows.append({
            'heldout_fold': heldout_fold,
            'sample_index': positive_count + local_index,
            'sample_type': 'topology_negative',
            'subject_id': subject_id,
            'subject_development_fold': int(
                target_by_subject[subject_id]['development_fold']
            ),
            'lesion_volume_bin': target_by_subject[subject_id][
                'effective_lesion_volume_bin'
            ],
            'supervision_mode': target_by_subject[subject_id]['supervision_mode'],
            'center_x': int(center[0]),
            'center_y': int(center[1]),
            'center_z': int(center[2]),
            'patch_sampling_support_voxels': 0,
            'patch_exclusion_support_voxels': int(exclusion[slices].sum()),
            'topology_relation': relation,
        })

    for local_index in range(brain_negative_count):
        subject_id, center, slices = choose_brain_negative(
            int(rng.integers(0, len(brain_pool))), brain_pool, rng
        )
        exclusion = unpack_mask(mask_records[subject_id]['exclusion_support'])
        sampling_rows.append({
            'heldout_fold': heldout_fold,
            'sample_index': positive_count + topology_negative_count + local_index,
            'sample_type': 'brain_negative',
            'subject_id': subject_id,
            'subject_development_fold': int(
                target_by_subject[subject_id]['development_fold']
            ),
            'lesion_volume_bin': target_by_subject[subject_id][
                'effective_lesion_volume_bin'
            ],
            'supervision_mode': target_by_subject[subject_id]['supervision_mode'],
            'center_x': int(center[0]),
            'center_y': int(center[1]),
            'center_z': int(center[2]),
            'patch_sampling_support_voxels': 0,
            'patch_exclusion_support_voxels': int(exclusion[slices].sum()),
            'topology_relation': '',
        })

    fold_rows = [row for row in sampling_rows if row['heldout_fold'] == heldout_fold]
    fold_schedule_df = pd.DataFrame(fold_rows)
    positive_subjects = set(fold_schedule_df.loc[
        fold_schedule_df['sample_type'] == 'positive', 'subject_id'
    ].astype(str))
    expected_positive_subjects = set(fold_train_df.loc[
        fold_train_df['trial8_sampling_positive'].map(as_bool), 'subject_id'
    ].astype(str))
    soft_only_count = int((
        (fold_schedule_df['sample_type'] == 'positive')
        & (fold_schedule_df['supervision_mode'] == 'soft_only_subvoxel')
    ).sum())
    schedule_summary_rows.append({
        'heldout_fold': heldout_fold,
        'fit_subjects': len(fold_train_df),
        'heldout_subjects': len(fold_heldout_df),
        'positive_patches': positive_count,
        'topology_negative_patches': topology_negative_count,
        'brain_negative_patches': brain_negative_count,
        'total_patches': len(fold_schedule_df),
        'positive_fraction': float(positive_count / len(fold_schedule_df)),
        'all_positive_subjects_covered': (
            expected_positive_subjects.issubset(positive_subjects)
        ),
        'heldout_subjects_sampled': int(
            fold_schedule_df['subject_id'].isin(
                fold_heldout_df['subject_id']
            ).sum()
        ),
        'soft_only_positive_patches': soft_only_count,
        'soft_only_case_in_training_partition': bool(
            (fold_train_df['trial8_supervision_mode'] == 'soft_only_subvoxel').any()
        ),
    })

sampling_audit_df = pd.DataFrame(sampling_rows)
schedule_summary_df = pd.DataFrame(schedule_summary_rows)
atomic_write_csv(sampling_audit_df, SAMPLING_AUDIT_PATH)
atomic_write_csv(schedule_summary_df, FOLD_SCHEDULE_SUMMARY_PATH)






soft_only_subject = target_audit_df.loc[
    target_audit_df['supervision_mode'] == 'soft_only_subvoxel', 'subject_id'
].iloc[0]
soft_only_components = []
for epoch in [0, 1]:
    rng = np.random.default_rng(SEED + epoch * 1_000_003)
    support = unpack_mask(mask_records[soft_only_subject]['sampling_support'])
    labels, count = ndimage_label_26n(support)
    center, patch_voxels = choose_positive_center(
        soft_only_subject, rng, component_index=epoch
    )
    selected_component = int(labels[tuple(np.clip(
        center, 0, np.asarray(labels.shape) - 1
    ))])
    
    
    soft_only_components.append({
        'epoch': epoch,
        'available_components': count,
        'intended_component': 1 + (epoch % count),
        'center_component_after_jitter': selected_component,
        'patch_support_voxels': patch_voxels,
    })


def soft_dice_per_case(logits, soft_target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = soft_target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def soft_focal_bce_per_case(logits, soft_target, gamma=2.0):
    logits = logits.float()
    target = soft_target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, raw.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def focal_tversky_per_case(
    logits, binary_target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = binary_target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error_floor = 1e-6
    return (
        (1.0 - tversky).clamp_min(error_floor).pow(gamma)
        - float(error_floor ** gamma)
    )


def revised_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    dice = soft_dice_per_case(logits, soft_target)
    focal_bce = soft_focal_bce_per_case(logits, soft_target)
    hard_tversky = focal_tversky_per_case(logits, binary_target)
    hard_contribution = 0.20 * hard_tversky * hard_weight.reshape(-1)
    total_per_case = 0.50 * dice + 0.30 * focal_bce + hard_contribution
    return total_per_case.mean(), {
        'soft_dice': dice.mean(),
        'soft_focal_bce': focal_bce.mean(),
        'hard_focal_tversky': hard_tversky.mean(),
        'hard_contribution': hard_contribution.mean(),
    }


def load_target_patch(subject_id, positive, component_index=None):
    row = train_by_subject[subject_id]
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask']) > 0
        hard_weight = scalar(data, 'binary_loss_weight')
    intended_component_mask = np.zeros_like(binary, dtype=bool)
    intended_component = 0
    intended_component_global_voxels = 0
    if positive:
        support = binary > 0
        if not support.any():
            support = soft > 0
        if component_index is None:
            center_support = support
        else:
            labels, component_count = ndimage_label_26n(support)
            if component_count < 1:
                raise RuntimeError(
                    f'{subject_id}: component probe has no target support.'
                )
            intended_component = 1 + (
                int(component_index) % int(component_count)
            )
            intended_component_mask = labels == intended_component
            intended_component_global_voxels = int(
                intended_component_mask.sum()
            )
            center_support = intended_component_mask
        center = np.rint(
            np.argwhere(center_support).mean(axis=0)
        ).astype(int)
    else:
        coordinates = np.argwhere(crop_mask)
        center = coordinates[len(coordinates) // 2]
    slices = patch_slices_from_center(binary.shape, center, PATCH_SIZE)
    return (
        extract_patch(binary, slices, PATCH_SIZE),
        extract_patch(soft, slices, PATCH_SIZE),
        hard_weight,
        extract_patch(
            intended_component_mask, slices, PATCH_SIZE
        ).astype(bool),
        intended_component,
        intended_component_global_voxels,
    )


standard_subject = target_audit_df[
    target_audit_df['supervision_mode'] == 'hard_and_soft'
].sort_values('effective_target_volume_ml').iloc[len(
    target_audit_df[target_audit_df['supervision_mode'] == 'hard_and_soft']
) // 2]['subject_id']
true_empty_subject = target_audit_df.loc[
    target_audit_df['supervision_mode'] == 'true_empty', 'subject_id'
].iloc[0]

loss_probe_cases = [
    ('hard_and_soft', str(standard_subject), True, None),
    ('soft_only_subvoxel', str(soft_only_subject), True, 0),
    ('soft_only_subvoxel', str(soft_only_subject), True, 1),
    ('true_empty', str(true_empty_subject), False, None),
]
loss_rows = []
for mode, subject_id, positive, component_index in loss_probe_cases:
    (
        binary_patch,
        soft_patch,
        hard_weight,
        intended_component_patch,
        intended_component,
        intended_component_global_voxels,
    ) = load_target_patch(
        subject_id, positive, component_index=component_index
    )
    logits = torch.full(
        (1, 1, *PATCH_SIZE), -2.0, dtype=torch.float32, requires_grad=True
    )
    soft_tensor = torch.from_numpy(soft_patch[None, None]).float()
    binary_tensor = torch.from_numpy(binary_patch[None, None]).float()
    hard_tensor = torch.as_tensor([hard_weight], dtype=torch.float32)
    loss, parts = revised_segmentation_loss(
        logits, soft_tensor, binary_tensor, hard_tensor
    )
    loss.backward()
    gradient = logits.grad.detach()
    support = soft_tensor > 0
    support_gradient = gradient[support]
    background_gradient = gradient[~support]
    intended_component_tensor = torch.from_numpy(
        intended_component_patch[None, None]
    )
    intended_component_gradient = gradient[intended_component_tensor]
    loss_rows.append({
        'supervision_mode': mode,
        'subject_id': subject_id,
        'component_probe_index': (
            -1 if component_index is None else int(component_index)
        ),
        'intended_component': int(intended_component),
        'intended_component_global_voxels': int(
            intended_component_global_voxels
        ),
        'intended_component_patch_voxels': int(
            intended_component_tensor.sum()
        ),
        'intended_component_voxels_with_nonzero_gradient': int(
            (intended_component_gradient.abs() > 0).sum()
        ),
        'binary_voxels': int(binary_patch.sum()),
        'soft_sum': float(soft_patch.sum(dtype=np.float64)),
        'hard_loss_weight': hard_weight,
        'total_loss': float(loss.detach()),
        'soft_dice_loss': float(parts['soft_dice'].detach()),
        'soft_focal_bce': float(parts['soft_focal_bce'].detach()),
        'hard_focal_tversky': float(parts['hard_focal_tversky'].detach()),
        'hard_contribution': float(parts['hard_contribution'].detach()),
        'gradient_finite': bool(torch.isfinite(gradient).all()),
        'gradient_norm': float(torch.linalg.vector_norm(gradient)),
        'support_voxels_with_nonzero_gradient': int(
            (support_gradient.abs() > 0).sum()
        ),
        'support_voxels': int(support.sum()),
        'mean_absolute_support_gradient': (
            float(support_gradient.abs().mean()) if support.any() else np.nan
        ),
        'mean_absolute_background_gradient': float(
            background_gradient.abs().mean()
        ),
    })

loss_audit_df = pd.DataFrame(loss_rows)
atomic_write_csv(loss_audit_df, LOSS_AUDIT_PATH)






scaler_checks = {
    'five_fold_scalers_written': len(scaler_audit_df) == NUMBER_OF_FOLDS,
    'fold_scaler_fit_counts_match': bool(
        scaler_audit_df['fit_subjects'].eq(
            scaler_audit_df['expected_fit_subjects']
        ).all()
    ),
    'fold_scaler_fit_and_heldout_disjoint': bool(
        scaler_audit_df['fit_and_heldout_disjoint'].all()
    ),
    'heldout_sidecars_not_read_for_own_scaler': not bool(
        scaler_audit_df['heldout_sidecars_read_for_scaler'].any()
    ),
    'targets_not_read_for_scalers': not bool(
        scaler_audit_df['target_arrays_read_for_scaler'].any()
    ),
    'all_fold_scaler_features_supported': bool(
        scaler_audit_df[[
            'territory_features_supported', 'vessel_features_supported',
            'case_features_supported',
        ]].all().all()
    ),
    'all_fold_scaled_features_nonconstant': bool(
        scaler_audit_df[[
            'territory_scaled_features_nonconstant',
            'vessel_scaled_features_nonconstant',
            'case_scaled_features_nonconstant',
        ]].all().all()
    ),
    'all_fold_scaler_statistics_finite': bool(
        scaler_audit_df['all_statistics_finite'].all()
    ),
    'all_fold_scaler_stds_positive': bool(
        scaler_audit_df['all_scaled_stds_positive'].all()
    ),
}

target_checks = {
    'exactly_102_train_targets_rechecked': len(target_audit_df) == 102,
    'no_locked_validation_or_test_targets_read': True,
    'exactly_one_soft_only_subvoxel_case': int(
        (target_audit_df['supervision_mode'] == 'soft_only_subvoxel').sum()
    ) == 1,
    'exactly_one_true_empty_train_case': int(
        (target_audit_df['supervision_mode'] == 'true_empty').sum()
    ) == 1,
    'soft_only_case_has_five_sampling_support_voxels': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'sampling_support_voxels',
        ].eq(5).all()
    ),
    'soft_only_case_hard_loss_disabled': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'binary_loss_weight',
        ].eq(0.0).all()
    ),
    'soft_only_case_excluded_from_hard_metrics': not bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'hard_metric_eligible',
        ].any()
    ),
    'soft_only_case_has_no_hard_target': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'binary_voxels',
        ].eq(0).all()
    ),
    'soft_only_case_retains_soft_target': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
            'soft_nonzero_voxels',
        ].eq(5).all()
    ),
    'hard_and_soft_cases_keep_hard_loss': bool(
        target_audit_df.loc[
            target_audit_df['supervision_mode'] == 'hard_and_soft',
            'binary_loss_weight',
        ].eq(1.0).all()
    ),
    
    
    
    
    'sampling_mask_overlap_fractions_are_valid': bool(
        target_audit_df['support_inside_crop_fraction'].between(0.0, 1.0).all()
    ),
}

positive_df = sampling_audit_df[
    sampling_audit_df['sample_type'] == 'positive'
]
negative_df = sampling_audit_df[
    sampling_audit_df['sample_type'].isin(
        ['topology_negative', 'brain_negative']
    )
]
sampling_checks = {
    'five_fold_schedules_created': len(schedule_summary_df) == NUMBER_OF_FOLDS,
    'no_heldout_subject_sampled': bool(
        schedule_summary_df['heldout_subjects_sampled'].eq(0).all()
    ),
    'every_fold_train_positive_subject_covered': bool(
        schedule_summary_df['all_positive_subjects_covered'].all()
    ),
    'positive_fraction_between_0p60_and_0p70': bool(
        schedule_summary_df['positive_fraction'].between(0.60, 0.70).all()
    ),
    'all_positive_patches_retain_sampling_support': bool(
        positive_df['patch_sampling_support_voxels'].gt(0).all()
    ),
    'all_negative_patches_exclude_hard_and_soft_support': bool(
        negative_df['patch_exclusion_support_voxels'].eq(0).all()
    ),
    'all_topology_negatives_have_relation': bool(
        sampling_audit_df.loc[
            sampling_audit_df['sample_type'] == 'topology_negative',
            'topology_relation',
        ].astype(str).str.len().gt(0).all()
    ),
    'soft_only_case_never_used_as_negative': not bool(
        negative_df['supervision_mode'].eq('soft_only_subvoxel').any()
    ),
    'soft_only_case_at_most_once_per_training_epoch': bool(
        schedule_summary_df['soft_only_positive_patches'].le(
            MAXIMUM_SOFT_ONLY_PATCHES_PER_EPOCH
        ).all()
    ),
    'soft_only_case_sampled_once_exactly_when_in_training_partition': bool(
        schedule_summary_df['soft_only_positive_patches'].eq(
            schedule_summary_df[
                'soft_only_case_in_training_partition'
            ].astype(int)
        ).all()
    ),
    'every_positive_volume_bin_has_at_least_twenty_patches_per_fold': bool(
        positive_df.groupby(
            ['heldout_fold', 'lesion_volume_bin']
        ).size().reindex(
            pd.MultiIndex.from_product([
                range(NUMBER_OF_FOLDS),
                ['tiny', 'small', 'medium', 'large', 'very_large'],
            ]),
            fill_value=0,
        ).ge(MINIMUM_POSITIVE_PATCHES_PER_BIN).all()
    ),
    'soft_only_two_components_alternate_across_two_epochs': (
        len(soft_only_components) == 2
        and {record['intended_component'] for record in soft_only_components}
        == {1, 2}
        and all(record['patch_support_voxels'] > 0 for record in soft_only_components)
        and sum(
            record['patch_support_voxels'] for record in soft_only_components
        ) == int(
            target_audit_df.loc[
                target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
                'sampling_support_voxels',
            ].iloc[0]
        )
    ),
}

soft_loss_rows = loss_audit_df[
    loss_audit_df['supervision_mode'] == 'soft_only_subvoxel'
].copy()
soft_only_global_support_voxels = int(
    target_audit_df.loc[
        target_audit_df['supervision_mode'] == 'soft_only_subvoxel',
        'sampling_support_voxels',
    ].iloc[0]
)
loss_checks = {
    'three_supervision_modes_smoke_tested': set(
        loss_audit_df['supervision_mode']
    ) == {'hard_and_soft', 'soft_only_subvoxel', 'true_empty'},
    'all_probe_losses_finite': bool(
        np.isfinite(loss_audit_df[[
            'total_loss', 'soft_dice_loss', 'soft_focal_bce',
            'hard_focal_tversky', 'hard_contribution',
        ]].to_numpy(dtype=np.float64)).all()
    ),
    'all_probe_gradients_finite_and_nonzero': bool(
        loss_audit_df['gradient_finite'].all()
        and loss_audit_df['gradient_norm'].gt(0).all()
    ),
    'soft_only_hard_contribution_exactly_zero': bool(
        np.isclose(
            soft_loss_rows['hard_contribution'].to_numpy(dtype=np.float64),
            0.0,
            atol=1e-12,
            rtol=0.0,
        ).all()
    ),
    'soft_only_all_component_support_voxels_receive_gradient': (
        len(soft_loss_rows) == 2
        and set(soft_loss_rows['intended_component'].astype(int)) == {1, 2}
        and bool(soft_loss_rows[
            'intended_component_patch_voxels'
        ].eq(soft_loss_rows['intended_component_global_voxels']).all())
        and bool(soft_loss_rows[
            'intended_component_voxels_with_nonzero_gradient'
        ].eq(soft_loss_rows['intended_component_patch_voxels']).all())
        and int(soft_loss_rows[
            'intended_component_global_voxels'
        ].sum()) == soft_only_global_support_voxels
    ),
    'soft_only_support_gradient_exceeds_background_gradient': bool(
        soft_loss_rows['mean_absolute_support_gradient'].gt(
            soft_loss_rows['mean_absolute_background_gradient']
        ).all()
    ),
}

all_checks = {
    **upstream_checks,
    **fold_cross_checks,
    **manifest_checks,
    **scaler_checks,
    **target_checks,
    **sampling_checks,
    **loss_checks,
    'architecture_parameter_count_frozen': True,
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
    'training_not_performed': True,
    'study_trainable_parameters_are_zero': True,
}
preflight_passed = bool(all(all_checks.values()))

training_policy = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'policy_version': PREFLIGHT_VERSION,
    'baseline_scientific_core': (
        'unchanged 15-channel residual 3D U-Net plus heterogeneous '
        'relation-aware GATv2 and bounded zero-init multiscale FiLM'
    ),
    'architecture_changed_in_this_preflight': False,
    'architecture_revision_required_before_training': True,
    'planned_architecture_revision': (
        'asymmetric lightweight vascular stream plus confidence-gated '
        'residual fusion, CTA/Tmax-informed occlusion source, and a small '
        'pretreatment clinical token'
    ),
    'total_parameters': EXPECTED_PARAMETERS,
    'development_design': 'five_fold_cross_validation_with_locked_confirmation',
    'fold_count': NUMBER_OF_FOLDS,
    'fold_scalers': fold_scaler_paths,
    'fold_scaler_sha256': fold_scaler_hashes,
    'patch_size': list(PATCH_SIZE),
    'input_channels': 15,
    'positive_sampling': (
        'all development-train positive subjects once; each lesion-volume '
        'bin padded to at least 20 using hard-target cases only'
    ),
    'minimum_positive_patches_per_bin': MINIMUM_POSITIVE_PATCHES_PER_BIN,
    'soft_only_subvoxel_policy': {
        'positive_center_mask': 'target_soft > 0',
        'maximum_samples_per_epoch': MAXIMUM_SOFT_ONLY_PATCHES_PER_EPOCH,
        'component_selection': 'alternate connected components by epoch',
        'hard_loss_weight': 0.0,
        'hard_metric_eligible': False,
        'negative_sampling_eligible': False,
    },
    'negative_sampling': (
        'topology and brain negatives require zero voxels from '
        'target_binary OR target_soft>0 support'
    ),
    'epoch_resampling': (
        'rebuild deterministically from fold and epoch seeds; rotate repeated '
        'hard-target subjects so padding does not repeatedly favor the same cases'
    ),
    'positive_fraction_target': [0.60, 0.70],
    'volume_case_loss_weighting': False,
    'clinical_window_recalibration': False,
    'frozen_input_channels_used_unchanged': True,
    'loss': {
        'soft_dice_weight': 0.50,
        'soft_focal_bce_weight': 0.30,
        'hard_focal_tversky_weight': 0.20,
        'hard_focal_tversky_alpha_false_positive': 0.30,
        'hard_focal_tversky_beta_false_negative': 0.70,
        'hard_focal_tversky_gamma': 0.75,
        'top_k_ohem': False,
    },
    'deep_supervision_weights': {
        'half_resolution': 0.15,
        'quarter_resolution': 0.10,
    },
    'soft_only_auxiliary_policy': (
        'main-resolution soft loss only; exclude from half/quarter losses to '
        'avoid erasing five-voxel subvoxel support during downsampling'
    ),
    'territory_node_loss_weight': 0.02,
    'territory_positive_weight': 4.0,
    'territory_node_supervision_policy': (
        'hard-target-eligible and node-valid subjects only; soft-only subvoxel '
        'case excluded because its empty hard mask is not a true negative'
    ),
    'batch_size': 1,
    'gradient_accumulation_steps': 4,
    'effective_batch_size': 4,
    'maximum_epochs': 60,
    'validation_interval_epochs': 1,
    'metrics_printed_every_epoch': True,
    'early_stopping_minimum_epoch': 20,
    'early_stopping_patience_validations': 15,
    'early_stopping_minimum_delta': 0.002,
    'optimizer': 'AdamW',
    'learning_rate': 8e-5,
    'weight_decay': 2e-3,
    'scheduler': 'three_epoch_linear_warmup_then_cosine',
    'gradient_clip_norm': 5.0,
    'mixed_precision': True,
    'ema_decay': 0.99,
    'validation_model': 'exponential_moving_average',
    'intensity_gain_range': [0.92, 1.08],
    'intensity_offset_range': [-0.02, 0.02],
    'noise_probability': 0.50,
    'maximum_noise_standard_deviation': 0.015,
    'gamma_probability': 0.40,
    'gamma_range': [0.90, 1.10],
    'spatial_affine_probability': 0.60,
    'spatial_rotation_degrees': 8.0,
    'spatial_scale_range': [0.92, 1.08],
    'spatial_translation_fraction_xy': 0.05,
    'spatial_translation_fraction_z': 0.025,
    'left_right_reflection': False,
    'post_augmentation_positive_support_required': True,
    'augmentation_retry_then_identity_fallback': True,
    'graph_feature_mask_probability': 0.05,
    'atlas_dropout_probability': 0.25,
    'checkpoint_metric': 'heldout_positive_mean_dice',
    'threshold_grid': [
        round(value, 2) for value in np.arange(0.10, 0.81, 0.05)
    ],
    'heldout_fold_metrics': [
        'positive_case_mean_dice', 'global_dice', 'lesion_precision',
        'lesion_recall', 'lesion_f1', 'absolute_volume_error_ml',
        'false_positive_volume_on_true_empty_ml',
    ],
    'validation_inference': 'full_volume_sliding_window_not_sampled_patches',
    'hard_metric_exclusion': ['soft_only_subvoxel'],
    'locked_validation_policy': (
        'do not read during five-fold development; use once only after '
        'training policy and hyperparameters are frozen'
    ),
    'test_policy': 'sealed until final evaluation',
    'baseline_loader_loss_sampler_contract_verified': preflight_passed,
    'fold0_pilot_training_authorized': False,
    'full_fivefold_training_authorized': False,
}
atomic_write_json(training_policy, TRAINING_POLICY_PATH)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': all_checks,
    'target_policy_manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'target_policy_manifest_sha256': target_policy_manifest_sha256,
    'fold_manifest_path': str(FOLD_MANIFEST_PATH),
    'fold_manifest_sha256': fold_manifest_sha256,
    'training_policy_path': str(TRAINING_POLICY_PATH),
    'fold_scaler_paths': fold_scaler_paths,
    'fold_scaler_sha256': fold_scaler_hashes,
    'target_audit_path': str(TARGET_AUDIT_PATH),
    'scaler_audit_path': str(SCALER_AUDIT_PATH),
    'sampling_audit_path': str(SAMPLING_AUDIT_PATH),
    'loss_audit_path': str(LOSS_AUDIT_PATH),
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'training_performed': False,
    'study_trainable_parameters': 0,
    'architecture_parameter_count': EXPECTED_PARAMETERS,
    'architecture_changed_in_this_preflight': False,
    'architecture_revision_required_before_training': True,
    'baseline_loader_loss_sampler_contract_verified': preflight_passed,
    'fold0_pilot_training_authorized': False,
    'full_fivefold_training_authorized': False,
    'next_block': (
        'revised vascular-fusion architecture feature-availability and '
        'test-time-input audit'
    ),
}
atomic_write_json(gate, PREFLIGHT_GATE_PATH)

print('\nFold-specific scaler audit:')
display(scaler_audit_df)
print('\nTrain target-policy recheck:')
display(target_audit_df.groupby(
    ['supervision_mode', 'effective_lesion_volume_bin'], as_index=False
).agg(
    subjects=('subject_id', 'count'),
    sampling_support_voxels=('sampling_support_voxels', 'sum'),
    minimum_effective_volume_ml=('effective_target_volume_ml', 'min'),
    maximum_effective_volume_ml=('effective_target_volume_ml', 'max'),
))
print('\nFive-fold sampling schedules:')
display(schedule_summary_df)
print('\nSoft-only component alternation:')
display(pd.DataFrame(soft_only_components))
print('\nReal-patch loss smoke audit:')
display(loss_audit_df)
print('\nPreflight checks:')
display(pd.DataFrame({
    'check': list(all_checks.keys()),
    'passed': list(all_checks.values()),
}))

if not preflight_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 FIVE-FOLD LOADER/LOSS/SAMPLER PREFLIGHT FAILED. '
        'Do not train. Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 FIVE-FOLD LOADER/LOSS/SAMPLER PREFLIGHT PASSED')
print('=' * 80)
print(f'Development subjects:               {len(train_df)}')
print(f'Fold-specific graph scalers:        {len(scaler_audit_df)}')
print(f'Patch size:                         {PATCH_SIZE}')
print('Volume-based case loss weights:     disabled')
print('Top-k OHEM:                         disabled')
print('Soft-only subvoxel supervision:     enabled')
print('Locked validation targets read:     False')
print('Test targets read:                  False')
print('Study trainable parameters:         0 (preflight only)')
print(f'Architecture parameters:            {EXPECTED_PARAMETERS:,} (unchanged)')
print(f'Baseline data contract verified:    {preflight_passed}')
print('Architecture revision required:     True')
print('Fold-0 pilot training authorized:   False')
print('Full five-fold training authorized: False')
print(f'Training policy:                    {TRAINING_POLICY_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print(
    '\nNext: run the revised vascular-fusion architecture feature-availability '
    'and test-time-input audit.'
)


In [ ]:
import sys
import os
import re
import json
import math
import hashlib
import subprocess
from datetime import datetime, time, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
ensure_package('openpyxl')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
RAW_DATA_ROOT = PROJECT_ROOT / 'ISLES24 dataset'
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
REVISED_MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AUDIT_VERSION = 'v1_feature_availability_and_test_time_contract'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_BASELINE_PARAMETERS = 15_052_628
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
    / 'graph_schema_verified_pickle_free.npz'
)
BASELINE_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
BASELINE_PREFLIGHT_GATE_PATH = (
    BASELINE_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / REVISED_MODEL_VERSION
    / AUDIT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CLINICAL_FEATURE_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_availability.csv'
CLINICAL_EXCLUSION_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_exclusions.csv'
SUBJECT_AUDIT_PATH = OUTPUT_ROOT / 'subject_input_availability.csv'
VASCULAR_AUDIT_PATH = OUTPUT_ROOT / 'vascular_feature_availability.csv'
ARCHITECTURE_CONTRACT_PATH = OUTPUT_ROOT / 'architecture_input_contract.json'
AUDIT_GATE_PATH = OUTPUT_ROOT / 'feature_availability_gate.json'

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
VASCULAR_IMAGE_CHANNELS = ['cta', 'tmax', 'perfusion_valid']
VASCULAR_PRIOR_ARRAYS = [
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_presence',
]

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    output = []
    for value in np.asarray(array).reshape(-1):
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def normalize_label(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).strip().lower())


def is_missing(value):
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def parse_binary(value):
    if is_missing(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        numeric = float(value)
        if numeric in {0.0, 1.0}:
            return numeric
        raise ValueError('binary value is outside 0/1')
    text = str(value).strip().lower()
    mapping = {
        '0': 0.0, 'no': 0.0, 'false': 0.0, 'n': 0.0,
        '1': 1.0, 'yes': 1.0, 'true': 1.0, 'y': 1.0,
    }
    if text not in mapping:
        raise ValueError('unrecognized binary value')
    return mapping[text]


def parse_sex(value):
    if is_missing(value):
        return None
    text = str(value).strip().lower()
    mapping = {'f': 1.0, 'female': 1.0, 'm': 0.0, 'male': 0.0}
    if text not in mapping:
        raise ValueError('unrecognized sex value')
    return mapping[text]


def parse_numeric(value):
    if is_missing(value):
        return None
    numeric = float(value)
    if not math.isfinite(numeric):
        raise ValueError('non-finite numeric value')
    return numeric


def parse_duration_hours(value):
    if is_missing(value):
        return None
    if isinstance(value, pd.Timedelta):
        hours = value.total_seconds() / 3600.0
    elif isinstance(value, time):
        hours = (
            value.hour + value.minute / 60.0 + value.second / 3600.0
            + value.microsecond / 3.6e9
        )
    elif isinstance(value, (int, float, np.integer, np.floating)):
        
        hours = float(value) * 24.0
    else:
        parsed = pd.to_timedelta(str(value).strip())
        hours = parsed.total_seconds() / 3600.0
    if not math.isfinite(hours) or hours < 0:
        raise ValueError('invalid duration')
    return hours


PARSER_BY_NAME = {
    'binary': parse_binary,
    'sex': parse_sex,
    'numeric': parse_numeric,
    'duration_hours': parse_duration_hours,
}


def read_baseline_table(path):
    suffix = Path(path).suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.tsv', '.txt'}:
        return pd.read_csv(path, sep='\t')
    raise RuntimeError(f'Unsupported baseline table format: {path}')



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'audit_version': AUDIT_VERSION,
        'feature_availability_audit_passed': False,
        'status': 'running_or_incomplete',
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
        'clinical_scaler_fitted': False,
        'study_model_constructed': False,
        'study_trainable_parameters': 0,
        'architecture_implementation_authorized': False,
        'model_training_authorized': False,
    },
    AUDIT_GATE_PATH,
)






required_paths = [
    PROJECT_ROOT,
    RAW_DATA_ROOT,
    TARGET_POLICY_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH,
    BASELINE_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 vascular-fusion audit prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)
schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
baseline_gate = json.loads(
    BASELINE_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)

manifest_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        'Target-policy manifest is missing columns: '
        + ', '.join(missing_manifest_columns)
    )
manifest_df['subject_id'] = (
    manifest_df['subject_id'].astype(str).str.strip().str.lower()
)
manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    schema_arrays = {name: np.asarray(schema[name]) for name in schema.files}
node_feature_names = decode_strings(schema_arrays['node_feature_names'])
vessel_feature_names = decode_strings(schema_arrays['vessel_feature_names'])
case_feature_names = decode_strings(schema_arrays['case_feature_names'])

upstream_checks = {
    'baseline_preflight_passed': baseline_gate.get('preflight_passed') is True,
    'baseline_data_contract_verified': (
        baseline_gate.get('baseline_loader_loss_sampler_contract_verified')
        is True
    ),
    'baseline_training_remained_locked': (
        baseline_gate.get('fold0_pilot_training_authorized') is False
        and baseline_gate.get('full_fivefold_training_authorized') is False
    ),
    'target_policy_manifest_hash_matches': (
        manifest_sha256 == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        schema_sha256 == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream contract failed: ' + ', '.join(failed))

print('Trial8 revised-architecture upstream contract passed.')
print(f'Manifest subjects:                   {len(manifest_df)}')
print(f'Split counts:                        {EXPECTED_SPLITS}')
print(f'Frozen manifest SHA-256:             {manifest_sha256}')
print(f'Pickle-free schema SHA-256:          {schema_sha256}')
print('Baseline training authorized:        False')
print('Target arrays read:                  False')
print('Outcome workbooks read:              False')
print('Study trainable parameters:          0 (availability audit only)')









CLINICAL_FEATURE_SPECS = [
    ('sex_female', ['Sex'], 'sex'),
    ('age_years', ['Age'], 'numeric'),
    ('atrial_fibrillation', ['Atrial fibrillation'], 'binary'),
    ('hypertension', ['Hypertension'], 'binary'),
    ('diabetes', ['Diabetes'], 'binary'),
    ('hyperlipidemia', ['Hyperlipidemia'], 'binary'),
    ('anticoagulation', ['Anticoagulation'], 'binary'),
    ('lipid_lowering_drugs', ['Lipid lowering drugs', 'Statins'], 'binary'),
    ('platelet_aggregation_inhibitors', ['PAIs'], 'binary'),
    ('glucose_mg_dl', ['Glucose'], 'numeric'),
    ('leucocytes_g_l', ['Leucocytes'], 'numeric'),
    ('crp_mg_dl', ['CRP'], 'numeric'),
    ('inr', ['INR'], 'numeric'),
    ('wake_up_stroke', ['Wake-up'], 'binary'),
    ('in_house_stroke', ['In-House', 'In-house'], 'binary'),
    ('external_referral', ['Referral'], 'binary'),
    ('onset_to_door_hours', ['Onset to door'], 'duration_hours'),
    ('alert_to_door_hours', ['Alert to door'], 'duration_hours'),
    ('nihss_at_admission', ['NIHSS at admission'], 'numeric'),
    ('mrs_at_admission', ['mRS at admission', 'MRS at admission'], 'numeric'),
    ('mrs_premorbid', ['mRS premorbid', 'MRS premorbid'], 'numeric'),
    ('door_to_imaging_hours', ['Door to imaging'], 'duration_hours'),
]

EXCLUDED_CLINICAL_SPECS = [
    ('Center', 'site_identifier_and_domain_shortcut'),
    ('Identifier', 'identifier'),
    ('Door to groin', 'post_imaging_treatment_timing'),
    ('Door to first series', 'post_imaging_treatment_timing'),
    ('Time of intervention', 'post_imaging_treatment_timing'),
    ('Door to recanalization', 'post_imaging_treatment_timing'),
    ('TICI postinterventional', 'post_treatment_outcome'),
    ('mTici postinterventional', 'post_treatment_outcome'),
    ('NIHSS 24h', 'post_treatment_outcome'),
    ('mRS 24h', 'post_treatment_outcome'),
    ('NIHSS discharge', 'post_treatment_outcome'),
    ('mRS discharge', 'post_treatment_outcome'),
    ('MRS discharge', 'post_treatment_outcome'),
    ('mRS 3 months', 'post_treatment_outcome'),
    ('MRS 3 months', 'post_treatment_outcome'),
]

selected_normalized_names = {
    normalize_label(alias)
    for _, aliases, _ in CLINICAL_FEATURE_SPECS
    for alias in aliases
}
excluded_normalized_names = {
    normalize_label(name) for name, _ in EXCLUDED_CLINICAL_SPECS
}
if selected_normalized_names & excluded_normalized_names:
    raise RuntimeError('Selected and excluded clinical fields overlap.')

clinical_exclusion_df = pd.DataFrame([
    {'source_field': name, 'exclusion_reason': reason, 'model_input': False}
    for name, reason in EXCLUDED_CLINICAL_SPECS
])
atomic_write_csv(clinical_exclusion_df, CLINICAL_EXCLUSION_AUDIT_PATH)






allowed_table_suffixes = {'.xlsx', '.xls', '.csv', '.tsv'}
baseline_candidates = sorted(
    path for path in RAW_DATA_ROOT.rglob('*demographic_baseline*')
    if path.is_file() and path.suffix.lower() in allowed_table_suffixes
)
manifest_subjects = set(manifest_df['subject_id'].astype(str))
baseline_by_subject = {}
unmapped_baseline_files = []
for path in baseline_candidates:
    match = re.search(r'(sub-stroke\d+)', str(path), flags=re.IGNORECASE)
    if match is None:
        unmapped_baseline_files.append(str(path))
        continue
    subject_id = match.group(1).lower()
    if subject_id not in manifest_subjects:
        continue
    if subject_id in baseline_by_subject:
        raise RuntimeError(
            f'{subject_id}: multiple baseline clinical workbooks found.'
        )
    baseline_by_subject[subject_id] = path

clinical_feature_records = {
    name: {
        'model_feature': name,
        'aliases': aliases,
        'parser': parser_name,
        'subjects_total': 0,
        'subjects_present': 0,
        'subjects_missing': 0,
        'parse_errors': 0,
        'train_present': 0,
        'val_present': 0,
        'test_present': 0,
        'resolved_source_columns': set(),
    }
    for name, aliases, parser_name in CLINICAL_FEATURE_SPECS
}

subject_clinical_rows = []
clinical_schema_sets = []
nihss_subscore_columns = set()
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id']).lower()
    split = str(manifest_row['dev_split'])
    baseline_path = baseline_by_subject.get(subject_id)
    if baseline_path is None:
        subject_clinical_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'baseline_clinical_path': '',
            'baseline_clinical_sha256': '',
            'baseline_rows': 0,
            'baseline_columns': 0,
            'selected_features_present': 0,
            'selected_features_missing': len(CLINICAL_FEATURE_SPECS),
            'clinical_parse_errors': 0,
            'aggregate_admission_nihss_available': False,
            'nihss_subscore_count': 0,
        })
        continue

    frame = read_baseline_table(baseline_path)
    frame = frame.dropna(axis=0, how='all').dropna(axis=1, how='all')
    if len(frame) != 1:
        raise RuntimeError(
            f'{subject_id}: expected one baseline row, observed {len(frame)}.'
        )
    row = frame.iloc[0]
    normalized_to_original = {
        normalize_label(column): str(column) for column in frame.columns
    }
    clinical_schema_sets.append(tuple(sorted(normalized_to_original)))
    subject_subscores = [
        str(column)
        for column in frame.columns
        if 'nihss' in normalize_label(column)
        and normalize_label(column) != normalize_label('NIHSS at admission')
    ]
    nihss_subscore_columns.update(subject_subscores)

    present_count = 0
    missing_count = 0
    parse_error_count = 0
    aggregate_nihss_available = False
    for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
        record = clinical_feature_records[model_name]
        record['subjects_total'] += 1
        source_column = next(
            (
                normalized_to_original[normalize_label(alias)]
                for alias in aliases
                if normalize_label(alias) in normalized_to_original
            ),
            None,
        )
        if source_column is None:
            record['subjects_missing'] += 1
            missing_count += 1
            continue
        record['resolved_source_columns'].add(source_column)
        value = row[source_column]
        try:
            parsed = PARSER_BY_NAME[parser_name](value)
        except Exception:
            parsed = None
            record['parse_errors'] += 1
            parse_error_count += 1
        if parsed is None:
            record['subjects_missing'] += 1
            missing_count += 1
        else:
            record['subjects_present'] += 1
            record[f'{split}_present'] += 1
            present_count += 1
            if model_name == 'nihss_at_admission':
                aggregate_nihss_available = True

    subject_clinical_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'baseline_clinical_path': str(baseline_path),
        'baseline_clinical_sha256': file_sha256(baseline_path),
        'baseline_rows': len(frame),
        'baseline_columns': len(frame.columns),
        'selected_features_present': present_count,
        'selected_features_missing': missing_count,
        'clinical_parse_errors': parse_error_count,
        'aggregate_admission_nihss_available': aggregate_nihss_available,
        'nihss_subscore_count': len(subject_subscores),
    })

clinical_rows = []
for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
    record = clinical_feature_records[model_name]
    clinical_rows.append({
        'model_feature': model_name,
        'source_aliases': ' | '.join(aliases),
        'resolved_source_columns': ' | '.join(
            sorted(record['resolved_source_columns'])
        ),
        'parser': parser_name,
        'subjects_total': record['subjects_total'],
        'subjects_present': record['subjects_present'],
        'subjects_missing': record['subjects_missing'],
        'missing_fraction': (
            record['subjects_missing'] / max(record['subjects_total'], 1)
        ),
        'parse_errors': record['parse_errors'],
        'train_present': record['train_present'],
        'val_present': record['val_present'],
        'test_present': record['test_present'],
        'paired_missingness_indicator': True,
        'scaler_policy': (
            'fit_on_development_fold_train_only_then_freeze'
            if parser_name in {'numeric', 'duration_hours'}
            else 'not_scaled_binary'
        ),
    })

clinical_feature_df = pd.DataFrame(clinical_rows)
clinical_subject_df = pd.DataFrame(subject_clinical_rows)
atomic_write_csv(clinical_feature_df, CLINICAL_FEATURE_AUDIT_PATH)






node_cta_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('cta_')
]
node_tmax_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('tmax_')
]
vessel_cta_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name in {'cta_mean', 'cta_p90'}
]
vessel_tmax_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name == 'tmax_mean'
]

vascular_rows = []
subject_vascular_rows = []
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id'])
    split = str(manifest_row['dev_split'])
    npz_path = Path(str(manifest_row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(manifest_row['trial8_graph_sidecar_path']))
    if not npz_path.exists():
        raise FileNotFoundError(f'{subject_id}: NPZ is missing: {npz_path}')
    if not sidecar_path.exists():
        raise FileNotFoundError(
            f'{subject_id}: graph sidecar is missing: {sidecar_path}'
        )
    if file_sha256(sidecar_path) != str(
        manifest_row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')

    with np.load(npz_path, allow_pickle=False) as data:
        npz_keys = set(data.files)
        missing_safe_keys = sorted(
            {'input_channels', *VASCULAR_PRIOR_ARRAYS} - npz_keys
        )
        input_channels = (
            decode_strings(data['input_channels'])
            if 'input_channels' in npz_keys
            else []
        )
        lvo_present = (
            bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            if 'lvo_present' in npz_keys
            else False
        )
        vessel_presence_count = (
            int((np.asarray(data['vessel_presence']) > 0).sum())
            if 'vessel_presence' in npz_keys
            else 0
        )

    with np.load(sidecar_path, allow_pickle=False) as data:
        sidecar_keys = set(data.files)
        forbidden_in_sidecar = sorted(FORBIDDEN_TARGET_KEYS & sidecar_keys)
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0

    atlas_enabled = as_bool(manifest_row['trial8_atlas_enabled'])
    valid_territory_nodes = int(node_valid.sum())
    valid_vessel_nodes = int(vessel_valid.sum())
    territory_cta_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_cta_indices]).sum()
    )
    territory_tmax_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_tmax_indices]).sum()
    )
    vessel_cta_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_cta_indices]).sum()
    )
    vessel_tmax_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_tmax_indices]).sum()
    )
    subject_vascular_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'input_channel_schema_matches': input_channels == EXPECTED_INPUT_CHANNELS,
        'cta_channel_available': 'cta' in input_channels,
        'tmax_channel_available': 'tmax' in input_channels,
        'ttp_channel_available': 'ttp' in input_channels,
        'perfusion_valid_channel_available': 'perfusion_valid' in input_channels,
        'cow_soft_union_available': 'cow_soft_union' in npz_keys,
        'lvo_proximity_available': 'lvo_proximity' in npz_keys,
        'lvo_present_field_available': 'lvo_present' in npz_keys,
        'lvo_present': lvo_present,
        'vessel_presence_field_available': 'vessel_presence' in npz_keys,
        'raw_vessel_presence_count': vessel_presence_count,
        'missing_safe_npz_keys': ' | '.join(missing_safe_keys),
        'sidecar_contains_forbidden_target_key': bool(forbidden_in_sidecar),
        'forbidden_sidecar_keys': ' | '.join(forbidden_in_sidecar),
        'atlas_enabled': atlas_enabled,
        'valid_territory_nodes': valid_territory_nodes,
        'valid_vessel_nodes': valid_vessel_nodes,
        'territory_cta_support': territory_cta_support,
        'territory_tmax_support': territory_tmax_support,
        'vessel_cta_support': vessel_cta_support,
        'vessel_tmax_support': vessel_tmax_support,
        'spatial_vascular_stream_available': (
            input_channels == EXPECTED_INPUT_CHANNELS
            and not missing_safe_keys
        ),
        'territory_graph_available': (
            atlas_enabled and valid_territory_nodes > 0
        ),
        'vessel_graph_available': valid_vessel_nodes > 0,
        'occlusion_source_valid': lvo_present,
    })

vascular_subject_df = pd.DataFrame(subject_vascular_rows)
for split, group in vascular_subject_df.groupby('dev_split', sort=True):
    vascular_rows.append({
        'dev_split': split,
        'subjects': len(group),
        'spatial_vascular_stream_available': int(
            group['spatial_vascular_stream_available'].sum()
        ),
        'territory_graph_available': int(
            group['territory_graph_available'].sum()
        ),
        'vessel_graph_available': int(group['vessel_graph_available'].sum()),
        'occlusion_source_valid': int(group['occlusion_source_valid'].sum()),
        'minimum_valid_territory_nodes': int(
            group['valid_territory_nodes'].min()
        ),
        'minimum_valid_vessel_nodes': int(group['valid_vessel_nodes'].min()),
        'mean_valid_vessel_nodes': float(group['valid_vessel_nodes'].mean()),
    })
vascular_audit_df = pd.DataFrame(vascular_rows)
atomic_write_csv(vascular_audit_df, VASCULAR_AUDIT_PATH)

subject_audit_df = clinical_subject_df.merge(
    vascular_subject_df,
    on=['subject_id', 'dev_split'],
    how='outer',
    validate='one_to_one',
)
atomic_write_csv(subject_audit_df, SUBJECT_AUDIT_PATH)






clinical_feature_count = len(CLINICAL_FEATURE_SPECS)
clinical_token_input_dim = clinical_feature_count * 2

architecture_contract = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'architecture_version': REVISED_MODEL_VERSION,
    'status': 'input_contract_only_model_not_constructed',
    'preserved_reference_path': {
        'image_backbone': '15-channel compact residual 3D U-Net',
        'graph_encoder': 'heterogeneous relation-aware GATv2',
        'graph_modulation': 'bounded zero-init multiscale FiLM',
        'reference_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    },
    'asymmetric_vascular_stream': {
        'spatial_inputs': VASCULAR_IMAGE_CHANNELS + [
            'cow_soft_union', 'lvo_proximity'
        ],
        'planned_channels': [8, 16, 32, 48],
        'planned_fusion_scales': ['quarter', 'eighth', 'sixteenth', 'bottleneck'],
        'fusion': 'confidence_gated_zero_initialized_residual_adapters',
        'confidence_inputs': [
            'perfusion_valid',
            'cow_prior_valid',
            'registration_confidence',
            'lvo_present',
        ],
        'purpose': (
            'emphasize CTA/perfusion/arterial structure without duplicating '
            'the full image backbone'
        ),
    },
    'occlusion_source': {
        'representation': 'validity_masked_source_token_or_graph_node',
        'inputs': [
            'lvo_present',
            'lvo_proximity',
            'vessel_cta_mean',
            'vessel_cta_p90',
            'vessel_tmax_mean',
            'vessel_minimum_lvo_distance_mm',
            'vessel_lvo_overlap_fraction',
        ],
        'ttp_used': False,
        'ttp_reason': 'TTP is not present in the frozen 15-channel input schema',
        'tmax_used': True,
        'invalid_state': 'zero token plus explicit validity bit',
        'external_deployment_dependency': (
            'requires the provided baseline LVO annotation or a separately '
            'validated CTA-derived occlusion detector'
        ),
    },
    'clinical_token': {
        'source': 'ses-01 demographic_baseline only',
        'selected_scalar_features': [
            name for name, _, _ in CLINICAL_FEATURE_SPECS
        ],
        'selected_scalar_count': clinical_feature_count,
        'missingness_indicator_count': clinical_feature_count,
        'input_dimension': clinical_token_input_dim,
        'planned_embedding_dimension': 32,
        'nihss_policy': 'aggregate admission NIHSS only; no subscores available',
        'missing_data_policy': (
            'fold-train median for continuous features, neutral zero for '
            'binary features, and one explicit missingness bit per scalar'
        ),
        'scaler_policy': 'fit inside each development fold train partition only',
        'excluded_fields': clinical_exclusion_df.to_dict('records'),
    },
    'anti_leakage': {
        'outcome_workbooks_read': False,
        'target_arrays_read': False,
        'center_used': False,
        'post_imaging_treatment_times_used': False,
        'post_treatment_outcomes_used': False,
        'clinical_scaler_fitted_in_this_audit': False,
    },
    'regularization_contract': {
        'zero_initialized_new_fusion_paths': True,
        'independent_vascular_branch_dropout': True,
        'clinical_token_dropout': True,
        'atlas_and_occlusion_source_dropout': True,
        'missing_inputs_use_explicit_validity_masks': True,
        'direct_graph_to_logit_residual': False,
        'expert_routing': False,
    },
    'parameter_budget': {
        'baseline_parameters': EXPECTED_BASELINE_PARAMETERS,
        'planned_total_minimum': 15_300_000,
        'planned_total_maximum': 15_650_000,
        'exact_count_required_in_next_model_preflight': True,
    },
    'architecture_implementation_authorized': True,
    'model_training_authorized': False,
}

clinical_checks = {
    'exactly_one_baseline_workbook_per_subject': (
        len(baseline_by_subject) == EXPECTED_SUBJECTS
        and clinical_subject_df['baseline_clinical_path'].astype(str).str.len().gt(0).all()
    ),
    'all_baseline_workbooks_have_one_row': bool(
        clinical_subject_df['baseline_rows'].eq(1).all()
    ),
    'baseline_clinical_schema_consistent': len(set(clinical_schema_sets)) == 1,
    'all_selected_features_have_train_support': bool(
        clinical_feature_df['train_present'].gt(0).all()
    ),
    'all_nonmissing_selected_values_parse': bool(
        clinical_feature_df['parse_errors'].eq(0).all()
    ),
    'aggregate_admission_nihss_available': bool(
        clinical_feature_df.loc[
            clinical_feature_df['model_feature'] == 'nihss_at_admission',
            'train_present',
        ].gt(0).all()
    ),
    'nihss_subscores_are_not_available_or_claimed': (
        len(nihss_subscore_columns) == 0
        and clinical_subject_df['nihss_subscore_count'].eq(0).all()
    ),
    'center_is_excluded': 'center' not in {
        name for name, _, _ in CLINICAL_FEATURE_SPECS
    },
    'post_imaging_treatment_fields_are_excluded': all(
        normalize_label(name) not in selected_normalized_names
        for name in [
            'Door to groin', 'Door to first series',
            'Time of intervention', 'Door to recanalization',
        ]
    ),
    'outcome_workbooks_not_read': True,
    'clinical_scaler_not_fitted': True,
}

vascular_checks = {
    'cta_tmax_and_perfusion_valid_channels_available_for_all': bool(
        vascular_subject_df[[
            'cta_channel_available',
            'tmax_channel_available',
            'perfusion_valid_channel_available',
        ]].all().all()
    ),
    'input_channel_schema_matches_for_all': bool(
        vascular_subject_df['input_channel_schema_matches'].all()
    ),
    'ttp_absent_and_tmax_selected': (
        not bool(vascular_subject_df['ttp_channel_available'].any())
        and bool(vascular_subject_df['tmax_channel_available'].all())
    ),
    'cow_and_lvo_prior_fields_available_for_all': bool(
        vascular_subject_df[[
            'cow_soft_union_available',
            'lvo_proximity_available',
            'lvo_present_field_available',
            'vessel_presence_field_available',
        ]].all().all()
    ),
    'spatial_vascular_stream_available_for_all': bool(
        vascular_subject_df['spatial_vascular_stream_available'].all()
    ),
    'graph_sidecars_contain_no_target_keys': not bool(
        vascular_subject_df['sidecar_contains_forbidden_target_key'].any()
    ),
    'all_atlas_enabled_subjects_have_territory_cta_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_cta_support'
        ].gt(0).all()
    ),
    'all_atlas_enabled_subjects_have_territory_tmax_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_tmax_support'
        ].gt(0).all()
    ),
    'vessel_cta_schema_present': len(vessel_cta_indices) == 2,
    'vessel_tmax_schema_present': len(vessel_tmax_indices) == 1,
    'missing_vascular_graph_inputs_have_validity_masks': bool(
        (
            (~vascular_subject_df['atlas_enabled'])
            | vascular_subject_df['territory_graph_available']
        ).all()
    ),
}

all_checks = {
    **upstream_checks,
    **clinical_checks,
    **vascular_checks,
    'clinical_token_dimension_is_44': clinical_token_input_dim == 44,
    'baseline_architecture_preserved_as_reference': True,
    'new_fusion_paths_planned_zero_initialized': True,
    'direct_graph_to_logit_residual_remains_disabled': True,
    'expert_routing_remains_disabled': True,
    'target_arrays_not_read': True,
    'locked_validation_and_test_inputs_used_for_availability_only': True,
    'study_model_not_constructed': True,
    'study_trainable_parameters_are_zero': True,
    'model_training_remains_locked': True,
}
audit_passed = bool(all(all_checks.values()))

architecture_contract['architecture_implementation_authorized'] = audit_passed
atomic_write_json(architecture_contract, ARCHITECTURE_CONTRACT_PATH)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'audit_version': AUDIT_VERSION,
    'feature_availability_audit_passed': audit_passed,
    'checks': all_checks,
    'manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'manifest_sha256': manifest_sha256,
    'graph_schema_path': str(GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': schema_sha256,
    'baseline_preflight_gate_path': str(BASELINE_PREFLIGHT_GATE_PATH),
    'clinical_feature_audit_path': str(CLINICAL_FEATURE_AUDIT_PATH),
    'clinical_exclusion_audit_path': str(CLINICAL_EXCLUSION_AUDIT_PATH),
    'vascular_feature_audit_path': str(VASCULAR_AUDIT_PATH),
    'subject_audit_path': str(SUBJECT_AUDIT_PATH),
    'architecture_contract_path': str(ARCHITECTURE_CONTRACT_PATH),
    'selected_clinical_scalar_features': clinical_feature_count,
    'clinical_token_input_dimension': clinical_token_input_dim,
    'nihss_subscores_available': False,
    'aggregate_admission_nihss_used': True,
    'ttp_available': False,
    'tmax_used_for_occlusion_source': True,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'clinical_scaler_fitted': False,
    'study_model_constructed': False,
    'study_trainable_parameters': 0,
    'architecture_implementation_authorized': audit_passed,
    'model_training_authorized': False,
    'next_block': (
        'construct and smoke-test VascuFateNet-v2 with exact parameter, '
        'neutrality, ablation, gradient, and L4-memory checks'
    ),
}
atomic_write_json(gate, AUDIT_GATE_PATH)

print('\nPretreatment clinical feature availability:')
display(clinical_feature_df)
print('\nExplicitly excluded clinical fields:')
display(clinical_exclusion_df)
print('\nVascular availability by split:')
display(vascular_audit_df)
print('\nFeature-availability checks:')
display(pd.DataFrame({
    'check': list(all_checks.keys()),
    'passed': list(all_checks.values()),
}))

if not audit_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT FAILED. '
        'Do not construct or train the revised model. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT PASSED')
print('=' * 80)
print(f'Subjects audited:                    {len(subject_audit_df)}')
print(f'Pretreatment clinical scalars:       {clinical_feature_count}')
print(f'Clinical missingness indicators:     {clinical_feature_count}')
print(f'Clinical-token input dimension:      {clinical_token_input_dim}')
print('Admission NIHSS:                     aggregate score available')
print('NIHSS subscores:                     unavailable; not used')
print('Occlusion timing signal:             Tmax (TTP unavailable)')
print('LVO source:                          validity-masked baseline annotation')
print('Outcome workbooks read:              False')
print('Target arrays read:                  False')
print('Clinical scaler fitted:              False')
print('Study trainable parameters:          0 (availability audit only)')
print(f'Baseline parameters preserved:       {EXPECTED_BASELINE_PARAMETERS:,}')
print(f'Architecture implementation allowed: {audit_passed}')
print('Model training authorized:           False')
print(f'Architecture contract:               {ARCHITECTURE_CONTRACT_PATH}')
print(f'Availability gate:                   {AUDIT_GATE_PATH}')
print(
    '\nNext: construct and smoke-test VascuFateNet-v2 with exact parameter, '
    'neutrality, ablation, gradient, and L4-memory checks.'
)


In [ ]:
import sys
import os
import re
import json
import math
import hashlib
import subprocess
from datetime import datetime, time, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
ensure_package('openpyxl')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
RAW_DATA_ROOT = PROJECT_ROOT / 'ISLES24 dataset'
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
REVISED_MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AUDIT_VERSION = 'v1_feature_availability_and_test_time_contract'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_BASELINE_PARAMETERS = 15_052_628
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
    / 'graph_schema_verified_pickle_free.npz'
)
BASELINE_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
BASELINE_PREFLIGHT_GATE_PATH = (
    BASELINE_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / REVISED_MODEL_VERSION
    / AUDIT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CLINICAL_FEATURE_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_availability.csv'
CLINICAL_EXCLUSION_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_exclusions.csv'
SUBJECT_AUDIT_PATH = OUTPUT_ROOT / 'subject_input_availability.csv'
VASCULAR_AUDIT_PATH = OUTPUT_ROOT / 'vascular_feature_availability.csv'
ARCHITECTURE_CONTRACT_PATH = OUTPUT_ROOT / 'architecture_input_contract.json'
AUDIT_GATE_PATH = OUTPUT_ROOT / 'feature_availability_gate.json'

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
VASCULAR_IMAGE_CHANNELS = ['cta', 'tmax', 'perfusion_valid']
VASCULAR_PRIOR_ARRAYS = [
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_presence',
]

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    output = []
    for value in np.asarray(array).reshape(-1):
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def normalize_label(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).strip().lower())


def is_missing(value):
    if value is None:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def parse_binary(value):
    if is_missing(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        numeric = float(value)
        if numeric in {0.0, 1.0}:
            return numeric
        raise ValueError('binary value is outside 0/1')
    text = str(value).strip().lower()
    mapping = {
        '0': 0.0, 'no': 0.0, 'false': 0.0, 'n': 0.0,
        '1': 1.0, 'yes': 1.0, 'true': 1.0, 'y': 1.0,
    }
    if text not in mapping:
        raise ValueError('unrecognized binary value')
    return mapping[text]


def parse_sex(value):
    if is_missing(value):
        return None
    text = str(value).strip().lower()
    mapping = {'f': 1.0, 'female': 1.0, 'm': 0.0, 'male': 0.0}
    if text not in mapping:
        raise ValueError('unrecognized sex value')
    return mapping[text]


def parse_numeric(value):
    if is_missing(value):
        return None
    numeric = float(value)
    if not math.isfinite(numeric):
        raise ValueError('non-finite numeric value')
    return numeric


def parse_duration_hours(value):
    if is_missing(value):
        return None
    if isinstance(value, pd.Timedelta):
        hours = value.total_seconds() / 3600.0
    elif isinstance(value, time):
        hours = (
            value.hour + value.minute / 60.0 + value.second / 3600.0
            + value.microsecond / 3.6e9
        )
    elif isinstance(value, (int, float, np.integer, np.floating)):
        
        hours = float(value) * 24.0
    else:
        parsed = pd.to_timedelta(str(value).strip())
        hours = parsed.total_seconds() / 3600.0
    if not math.isfinite(hours) or hours < 0:
        raise ValueError('invalid duration')
    return hours


PARSER_BY_NAME = {
    'binary': parse_binary,
    'sex': parse_sex,
    'numeric': parse_numeric,
    'duration_hours': parse_duration_hours,
}


def read_baseline_table(path):
    suffix = Path(path).suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.tsv', '.txt'}:
        return pd.read_csv(path, sep='\t')
    raise RuntimeError(f'Unsupported baseline table format: {path}')



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'audit_version': AUDIT_VERSION,
        'feature_availability_audit_passed': False,
        'status': 'running_or_incomplete',
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
        'clinical_scaler_fitted': False,
        'study_model_constructed': False,
        'study_trainable_parameters': 0,
        'architecture_implementation_authorized': False,
        'model_training_authorized': False,
    },
    AUDIT_GATE_PATH,
)






required_paths = [
    PROJECT_ROOT,
    RAW_DATA_ROOT,
    TARGET_POLICY_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH,
    BASELINE_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 vascular-fusion audit prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)
schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
baseline_gate = json.loads(
    BASELINE_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)

manifest_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        'Target-policy manifest is missing columns: '
        + ', '.join(missing_manifest_columns)
    )
manifest_df['subject_id'] = (
    manifest_df['subject_id'].astype(str).str.strip().str.lower()
)
manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    schema_arrays = {name: np.asarray(schema[name]) for name in schema.files}
node_feature_names = decode_strings(schema_arrays['node_feature_names'])
vessel_feature_names = decode_strings(schema_arrays['vessel_feature_names'])
case_feature_names = decode_strings(schema_arrays['case_feature_names'])

upstream_checks = {
    'baseline_preflight_passed': baseline_gate.get('preflight_passed') is True,
    'baseline_data_contract_verified': (
        baseline_gate.get('baseline_loader_loss_sampler_contract_verified')
        is True
    ),
    'baseline_training_remained_locked': (
        baseline_gate.get('fold0_pilot_training_authorized') is False
        and baseline_gate.get('full_fivefold_training_authorized') is False
    ),
    'target_policy_manifest_hash_matches': (
        manifest_sha256 == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        schema_sha256 == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream contract failed: ' + ', '.join(failed))

print('Trial8 revised-architecture upstream contract passed.')
print(f'Manifest subjects:                   {len(manifest_df)}')
print(f'Split counts:                        {EXPECTED_SPLITS}')
print(f'Frozen manifest SHA-256:             {manifest_sha256}')
print(f'Pickle-free schema SHA-256:          {schema_sha256}')
print('Baseline training authorized:        False')
print('Target arrays read:                  False')
print('Outcome workbooks read:              False')
print('Study trainable parameters:          0 (availability audit only)')









CLINICAL_FEATURE_SPECS = [
    ('sex_female', ['Sex'], 'sex'),
    ('age_years', ['Age'], 'numeric'),
    ('atrial_fibrillation', ['Atrial fibrillation'], 'binary'),
    ('hypertension', ['Hypertension'], 'binary'),
    ('diabetes', ['Diabetes'], 'binary'),
    ('hyperlipidemia', ['Hyperlipidemia'], 'binary'),
    ('anticoagulation', ['Anticoagulation'], 'binary'),
    ('lipid_lowering_drugs', ['Lipid lowering drugs', 'Statins'], 'binary'),
    ('platelet_aggregation_inhibitors', ['PAIs'], 'binary'),
    ('glucose_mg_dl', ['Glucose'], 'numeric'),
    ('leucocytes_g_l', ['Leucocytes'], 'numeric'),
    ('crp_mg_dl', ['CRP'], 'numeric'),
    ('inr', ['INR'], 'numeric'),
    ('wake_up_stroke', ['Wake-up'], 'binary'),
    ('in_house_stroke', ['In-House', 'In-house'], 'binary'),
    ('external_referral', ['Referral'], 'binary'),
    ('onset_to_door_hours', ['Onset to door'], 'duration_hours'),
    ('alert_to_door_hours', ['Alert to door'], 'duration_hours'),
    ('nihss_at_admission', ['NIHSS at admission'], 'numeric'),
    ('mrs_at_admission', ['mRS at admission', 'MRS at admission'], 'numeric'),
    ('mrs_premorbid', ['mRS premorbid', 'MRS premorbid'], 'numeric'),
    ('door_to_imaging_hours', ['Door to imaging'], 'duration_hours'),
]

EXCLUDED_CLINICAL_SPECS = [
    ('Center', 'site_identifier_and_domain_shortcut'),
    ('Identifier', 'identifier'),
    ('Door to groin', 'post_imaging_treatment_timing'),
    ('Door to first series', 'post_imaging_treatment_timing'),
    ('Time of intervention', 'post_imaging_treatment_timing'),
    ('Door to recanalization', 'post_imaging_treatment_timing'),
    ('TICI postinterventional', 'post_treatment_outcome'),
    ('mTici postinterventional', 'post_treatment_outcome'),
    ('NIHSS 24h', 'post_treatment_outcome'),
    ('mRS 24h', 'post_treatment_outcome'),
    ('NIHSS discharge', 'post_treatment_outcome'),
    ('mRS discharge', 'post_treatment_outcome'),
    ('MRS discharge', 'post_treatment_outcome'),
    ('mRS 3 months', 'post_treatment_outcome'),
    ('MRS 3 months', 'post_treatment_outcome'),
]

selected_normalized_names = {
    normalize_label(alias)
    for _, aliases, _ in CLINICAL_FEATURE_SPECS
    for alias in aliases
}
excluded_normalized_names = {
    normalize_label(name) for name, _ in EXCLUDED_CLINICAL_SPECS
}
if selected_normalized_names & excluded_normalized_names:
    raise RuntimeError('Selected and excluded clinical fields overlap.')


def resolve_baseline_row(frame, subject_id):
    
    observed_rows = len(frame)
    if observed_rows < 1:
        raise RuntimeError(f'{subject_id}: baseline workbook has no data row.')
    if observed_rows == 1:
        return frame.iloc[0], {
            'baseline_rows_observed': 1,
            'baseline_unique_rows': 1,
            'row_selection_method': 'single_row',
            'selected_source_row_index': str(frame.index[0]),
            'duplicate_rows_collapsed': 0,
        }

    
    
    
    unique_frame = frame.drop_duplicates(keep='first')
    if len(unique_frame) == 1:
        return unique_frame.iloc[0], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': 1,
            'row_selection_method': 'exact_duplicate_rows_collapsed',
            'selected_source_row_index': str(unique_frame.index[0]),
            'duplicate_rows_collapsed': observed_rows - 1,
        }

    normalized_to_original = {
        normalize_label(column): str(column) for column in unique_frame.columns
    }

    
    
    
    
    identifier_columns = [
        normalized_to_original[name]
        for name in ['identifier', 'subjectid', 'participantid', 'patientid']
        if name in normalized_to_original
    ]
    target_digits = ''.join(re.findall(r'\d+', subject_id)).lstrip('0') or '0'
    matched_indices = []
    for row_index, candidate in unique_frame.iterrows():
        for column in identifier_columns:
            value_digits = ''.join(re.findall(r'\d+', str(candidate[column])))
            value_digits = value_digits.lstrip('0') or '0'
            if value_digits == target_digits:
                matched_indices.append(row_index)
                break
    if len(matched_indices) == 1:
        selected_index = matched_indices[0]
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_subject_identifier_match',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }

    
    
    
    
    profiles = []
    for row_index, candidate in unique_frame.iterrows():
        valid_count = 0
        parse_errors = 0
        for _, aliases, parser_name in CLINICAL_FEATURE_SPECS:
            source_column = next(
                (
                    normalized_to_original[normalize_label(alias)]
                    for alias in aliases
                    if normalize_label(alias) in normalized_to_original
                ),
                None,
            )
            if source_column is None:
                continue
            try:
                parsed = PARSER_BY_NAME[parser_name](candidate[source_column])
            except Exception:
                parse_errors += 1
                continue
            if parsed is not None:
                valid_count += 1
        profiles.append({
            'row_index': row_index,
            'valid_selected_fields': valid_count,
            'parse_errors': parse_errors,
        })

    ordered = sorted(
        profiles,
        key=lambda item: (item['parse_errors'], -item['valid_selected_fields']),
    )
    best = ordered[0]
    runner_up = ordered[1]
    if (
        best['parse_errors'] == 0
        and best['valid_selected_fields']
        >= runner_up['valid_selected_fields'] + 3
    ):
        selected_index = best['row_index']
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_parse_complete_patient_row',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }

    profile_text = '; '.join(
        f"row={item['row_index']},valid={item['valid_selected_fields']},"
        f"errors={item['parse_errors']}"
        for item in profiles
    )
    raise RuntimeError(
        f'{subject_id}: {observed_rows} baseline rows remain ambiguous after '
        f'exact-deduplication ({len(unique_frame)} unique). Profiles: '
        f'{profile_text}. No row was selected.'
    )

clinical_exclusion_df = pd.DataFrame([
    {'source_field': name, 'exclusion_reason': reason, 'model_input': False}
    for name, reason in EXCLUDED_CLINICAL_SPECS
])
atomic_write_csv(clinical_exclusion_df, CLINICAL_EXCLUSION_AUDIT_PATH)






allowed_table_suffixes = {'.xlsx', '.xls', '.csv', '.tsv'}
baseline_candidates = sorted(
    path for path in RAW_DATA_ROOT.rglob('*demographic_baseline*')
    if path.is_file() and path.suffix.lower() in allowed_table_suffixes
)
manifest_subjects = set(manifest_df['subject_id'].astype(str))
baseline_by_subject = {}
unmapped_baseline_files = []
for path in baseline_candidates:
    match = re.search(r'(sub-stroke\d+)', str(path), flags=re.IGNORECASE)
    if match is None:
        unmapped_baseline_files.append(str(path))
        continue
    subject_id = match.group(1).lower()
    if subject_id not in manifest_subjects:
        continue
    if subject_id in baseline_by_subject:
        raise RuntimeError(
            f'{subject_id}: multiple baseline clinical workbooks found.'
        )
    baseline_by_subject[subject_id] = path

clinical_feature_records = {
    name: {
        'model_feature': name,
        'aliases': aliases,
        'parser': parser_name,
        'subjects_total': 0,
        'subjects_present': 0,
        'subjects_missing': 0,
        'parse_errors': 0,
        'train_present': 0,
        'val_present': 0,
        'test_present': 0,
        'resolved_source_columns': set(),
    }
    for name, aliases, parser_name in CLINICAL_FEATURE_SPECS
}

subject_clinical_rows = []
clinical_schema_sets = []
nihss_subscore_columns = set()
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id']).lower()
    split = str(manifest_row['dev_split'])
    baseline_path = baseline_by_subject.get(subject_id)
    if baseline_path is None:
        subject_clinical_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'baseline_clinical_path': '',
            'baseline_clinical_sha256': '',
            'baseline_rows_observed': 0,
            'baseline_unique_rows': 0,
            'row_selection_method': '',
            'selected_source_row_index': '',
            'duplicate_rows_collapsed': 0,
            'baseline_columns': 0,
            'selected_features_present': 0,
            'selected_features_missing': len(CLINICAL_FEATURE_SPECS),
            'clinical_parse_errors': 0,
            'aggregate_admission_nihss_available': False,
            'nihss_subscore_count': 0,
        })
        continue

    frame = read_baseline_table(baseline_path)
    frame = frame.dropna(axis=0, how='all').dropna(axis=1, how='all')
    row, row_resolution = resolve_baseline_row(frame, subject_id)
    normalized_to_original = {
        normalize_label(column): str(column) for column in frame.columns
    }
    clinical_schema_sets.append(tuple(sorted(normalized_to_original)))
    subject_subscores = [
        str(column)
        for column in frame.columns
        if 'nihss' in normalize_label(column)
        and normalize_label(column) != normalize_label('NIHSS at admission')
    ]
    nihss_subscore_columns.update(subject_subscores)

    present_count = 0
    missing_count = 0
    parse_error_count = 0
    aggregate_nihss_available = False
    for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
        record = clinical_feature_records[model_name]
        record['subjects_total'] += 1
        source_column = next(
            (
                normalized_to_original[normalize_label(alias)]
                for alias in aliases
                if normalize_label(alias) in normalized_to_original
            ),
            None,
        )
        if source_column is None:
            record['subjects_missing'] += 1
            missing_count += 1
            continue
        record['resolved_source_columns'].add(source_column)
        value = row[source_column]
        try:
            parsed = PARSER_BY_NAME[parser_name](value)
        except Exception:
            parsed = None
            record['parse_errors'] += 1
            parse_error_count += 1
        if parsed is None:
            record['subjects_missing'] += 1
            missing_count += 1
        else:
            record['subjects_present'] += 1
            record[f'{split}_present'] += 1
            present_count += 1
            if model_name == 'nihss_at_admission':
                aggregate_nihss_available = True

    subject_clinical_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'baseline_clinical_path': str(baseline_path),
        'baseline_clinical_sha256': file_sha256(baseline_path),
        **row_resolution,
        'baseline_columns': len(frame.columns),
        'selected_features_present': present_count,
        'selected_features_missing': missing_count,
        'clinical_parse_errors': parse_error_count,
        'aggregate_admission_nihss_available': aggregate_nihss_available,
        'nihss_subscore_count': len(subject_subscores),
    })

clinical_rows = []
for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
    record = clinical_feature_records[model_name]
    clinical_rows.append({
        'model_feature': model_name,
        'source_aliases': ' | '.join(aliases),
        'resolved_source_columns': ' | '.join(
            sorted(record['resolved_source_columns'])
        ),
        'parser': parser_name,
        'subjects_total': record['subjects_total'],
        'subjects_present': record['subjects_present'],
        'subjects_missing': record['subjects_missing'],
        'missing_fraction': (
            record['subjects_missing'] / max(record['subjects_total'], 1)
        ),
        'parse_errors': record['parse_errors'],
        'train_present': record['train_present'],
        'val_present': record['val_present'],
        'test_present': record['test_present'],
        'paired_missingness_indicator': True,
        'scaler_policy': (
            'fit_on_development_fold_train_only_then_freeze'
            if parser_name in {'numeric', 'duration_hours'}
            else 'not_scaled_binary'
        ),
    })

clinical_feature_df = pd.DataFrame(clinical_rows)
clinical_subject_df = pd.DataFrame(subject_clinical_rows)
atomic_write_csv(clinical_feature_df, CLINICAL_FEATURE_AUDIT_PATH)






node_cta_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('cta_')
]
node_tmax_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('tmax_')
]
vessel_cta_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name in {'cta_mean', 'cta_p90'}
]
vessel_tmax_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name == 'tmax_mean'
]

vascular_rows = []
subject_vascular_rows = []
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id'])
    split = str(manifest_row['dev_split'])
    npz_path = Path(str(manifest_row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(manifest_row['trial8_graph_sidecar_path']))
    if not npz_path.exists():
        raise FileNotFoundError(f'{subject_id}: NPZ is missing: {npz_path}')
    if not sidecar_path.exists():
        raise FileNotFoundError(
            f'{subject_id}: graph sidecar is missing: {sidecar_path}'
        )
    if file_sha256(sidecar_path) != str(
        manifest_row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')

    with np.load(npz_path, allow_pickle=False) as data:
        npz_keys = set(data.files)
        missing_safe_keys = sorted(
            {'input_channels', *VASCULAR_PRIOR_ARRAYS} - npz_keys
        )
        input_channels = (
            decode_strings(data['input_channels'])
            if 'input_channels' in npz_keys
            else []
        )
        lvo_present = (
            bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            if 'lvo_present' in npz_keys
            else False
        )
        vessel_presence_count = (
            int((np.asarray(data['vessel_presence']) > 0).sum())
            if 'vessel_presence' in npz_keys
            else 0
        )

    with np.load(sidecar_path, allow_pickle=False) as data:
        sidecar_keys = set(data.files)
        forbidden_in_sidecar = sorted(FORBIDDEN_TARGET_KEYS & sidecar_keys)
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0

    atlas_enabled = as_bool(manifest_row['trial8_atlas_enabled'])
    valid_territory_nodes = int(node_valid.sum())
    valid_vessel_nodes = int(vessel_valid.sum())
    territory_cta_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_cta_indices]).sum()
    )
    territory_tmax_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_tmax_indices]).sum()
    )
    vessel_cta_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_cta_indices]).sum()
    )
    vessel_tmax_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_tmax_indices]).sum()
    )
    subject_vascular_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'input_channel_schema_matches': input_channels == EXPECTED_INPUT_CHANNELS,
        'cta_channel_available': 'cta' in input_channels,
        'tmax_channel_available': 'tmax' in input_channels,
        'ttp_channel_available': 'ttp' in input_channels,
        'perfusion_valid_channel_available': 'perfusion_valid' in input_channels,
        'cow_soft_union_available': 'cow_soft_union' in npz_keys,
        'lvo_proximity_available': 'lvo_proximity' in npz_keys,
        'lvo_present_field_available': 'lvo_present' in npz_keys,
        'lvo_present': lvo_present,
        'vessel_presence_field_available': 'vessel_presence' in npz_keys,
        'raw_vessel_presence_count': vessel_presence_count,
        'missing_safe_npz_keys': ' | '.join(missing_safe_keys),
        'sidecar_contains_forbidden_target_key': bool(forbidden_in_sidecar),
        'forbidden_sidecar_keys': ' | '.join(forbidden_in_sidecar),
        'atlas_enabled': atlas_enabled,
        'valid_territory_nodes': valid_territory_nodes,
        'valid_vessel_nodes': valid_vessel_nodes,
        'territory_cta_support': territory_cta_support,
        'territory_tmax_support': territory_tmax_support,
        'vessel_cta_support': vessel_cta_support,
        'vessel_tmax_support': vessel_tmax_support,
        'spatial_vascular_stream_available': (
            input_channels == EXPECTED_INPUT_CHANNELS
            and not missing_safe_keys
        ),
        'territory_graph_available': (
            atlas_enabled and valid_territory_nodes > 0
        ),
        'vessel_graph_available': valid_vessel_nodes > 0,
        'occlusion_source_valid': lvo_present,
    })

vascular_subject_df = pd.DataFrame(subject_vascular_rows)
for split, group in vascular_subject_df.groupby('dev_split', sort=True):
    vascular_rows.append({
        'dev_split': split,
        'subjects': len(group),
        'spatial_vascular_stream_available': int(
            group['spatial_vascular_stream_available'].sum()
        ),
        'territory_graph_available': int(
            group['territory_graph_available'].sum()
        ),
        'vessel_graph_available': int(group['vessel_graph_available'].sum()),
        'occlusion_source_valid': int(group['occlusion_source_valid'].sum()),
        'minimum_valid_territory_nodes': int(
            group['valid_territory_nodes'].min()
        ),
        'minimum_valid_vessel_nodes': int(group['valid_vessel_nodes'].min()),
        'mean_valid_vessel_nodes': float(group['valid_vessel_nodes'].mean()),
    })
vascular_audit_df = pd.DataFrame(vascular_rows)
atomic_write_csv(vascular_audit_df, VASCULAR_AUDIT_PATH)

subject_audit_df = clinical_subject_df.merge(
    vascular_subject_df,
    on=['subject_id', 'dev_split'],
    how='outer',
    validate='one_to_one',
)
atomic_write_csv(subject_audit_df, SUBJECT_AUDIT_PATH)






clinical_feature_count = len(CLINICAL_FEATURE_SPECS)
clinical_token_input_dim = clinical_feature_count * 2

architecture_contract = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'architecture_version': REVISED_MODEL_VERSION,
    'status': 'input_contract_only_model_not_constructed',
    'preserved_reference_path': {
        'image_backbone': '15-channel compact residual 3D U-Net',
        'graph_encoder': 'heterogeneous relation-aware GATv2',
        'graph_modulation': 'bounded zero-init multiscale FiLM',
        'reference_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    },
    'asymmetric_vascular_stream': {
        'spatial_inputs': VASCULAR_IMAGE_CHANNELS + [
            'cow_soft_union', 'lvo_proximity'
        ],
        'planned_channels': [8, 16, 32, 48],
        'planned_fusion_scales': ['quarter', 'eighth', 'sixteenth', 'bottleneck'],
        'fusion': 'confidence_gated_zero_initialized_residual_adapters',
        'confidence_inputs': [
            'perfusion_valid',
            'cow_prior_valid',
            'registration_confidence',
            'lvo_present',
        ],
        'purpose': (
            'emphasize CTA/perfusion/arterial structure without duplicating '
            'the full image backbone'
        ),
    },
    'occlusion_source': {
        'representation': 'validity_masked_source_token_or_graph_node',
        'inputs': [
            'lvo_present',
            'lvo_proximity',
            'vessel_cta_mean',
            'vessel_cta_p90',
            'vessel_tmax_mean',
            'vessel_minimum_lvo_distance_mm',
            'vessel_lvo_overlap_fraction',
        ],
        'ttp_used': False,
        'ttp_reason': 'TTP is not present in the frozen 15-channel input schema',
        'tmax_used': True,
        'invalid_state': 'zero token plus explicit validity bit',
        'external_deployment_dependency': (
            'requires the provided baseline LVO annotation or a separately '
            'validated CTA-derived occlusion detector'
        ),
    },
    'clinical_token': {
        'source': 'ses-01 demographic_baseline only',
        'selected_scalar_features': [
            name for name, _, _ in CLINICAL_FEATURE_SPECS
        ],
        'selected_scalar_count': clinical_feature_count,
        'missingness_indicator_count': clinical_feature_count,
        'input_dimension': clinical_token_input_dim,
        'planned_embedding_dimension': 32,
        'nihss_policy': 'aggregate admission NIHSS only; no subscores available',
        'missing_data_policy': (
            'fold-train median for continuous features, neutral zero for '
            'binary features, and one explicit missingness bit per scalar'
        ),
        'scaler_policy': 'fit inside each development fold train partition only',
        'excluded_fields': clinical_exclusion_df.to_dict('records'),
    },
    'anti_leakage': {
        'outcome_workbooks_read': False,
        'target_arrays_read': False,
        'center_used': False,
        'post_imaging_treatment_times_used': False,
        'post_treatment_outcomes_used': False,
        'clinical_scaler_fitted_in_this_audit': False,
    },
    'regularization_contract': {
        'zero_initialized_new_fusion_paths': True,
        'independent_vascular_branch_dropout': True,
        'clinical_token_dropout': True,
        'atlas_and_occlusion_source_dropout': True,
        'missing_inputs_use_explicit_validity_masks': True,
        'direct_graph_to_logit_residual': False,
        'expert_routing': False,
    },
    'parameter_budget': {
        'baseline_parameters': EXPECTED_BASELINE_PARAMETERS,
        'planned_total_minimum': 15_300_000,
        'planned_total_maximum': 15_650_000,
        'exact_count_required_in_next_model_preflight': True,
    },
    'architecture_implementation_authorized': True,
    'model_training_authorized': False,
}

clinical_checks = {
    'exactly_one_baseline_workbook_per_subject': (
        len(baseline_by_subject) == EXPECTED_SUBJECTS
        and clinical_subject_df['baseline_clinical_path'].astype(str).str.len().gt(0).all()
    ),
    'all_baseline_workbooks_resolve_to_one_unambiguous_patient_row': bool(
        clinical_subject_df['row_selection_method'].fillna('').str.len().gt(0).all()
    ),
    'baseline_clinical_schema_consistent': len(set(clinical_schema_sets)) == 1,
    'all_selected_features_have_train_support': bool(
        clinical_feature_df['train_present'].gt(0).all()
    ),
    'all_nonmissing_selected_values_parse': bool(
        clinical_feature_df['parse_errors'].eq(0).all()
    ),
    'aggregate_admission_nihss_available': bool(
        clinical_feature_df.loc[
            clinical_feature_df['model_feature'] == 'nihss_at_admission',
            'train_present',
        ].gt(0).all()
    ),
    'nihss_subscores_are_not_available_or_claimed': (
        len(nihss_subscore_columns) == 0
        and clinical_subject_df['nihss_subscore_count'].eq(0).all()
    ),
    'center_is_excluded': 'center' not in {
        name for name, _, _ in CLINICAL_FEATURE_SPECS
    },
    'post_imaging_treatment_fields_are_excluded': all(
        normalize_label(name) not in selected_normalized_names
        for name in [
            'Door to groin', 'Door to first series',
            'Time of intervention', 'Door to recanalization',
        ]
    ),
    'outcome_workbooks_not_read': True,
    'clinical_scaler_not_fitted': True,
}

vascular_checks = {
    'cta_tmax_and_perfusion_valid_channels_available_for_all': bool(
        vascular_subject_df[[
            'cta_channel_available',
            'tmax_channel_available',
            'perfusion_valid_channel_available',
        ]].all().all()
    ),
    'input_channel_schema_matches_for_all': bool(
        vascular_subject_df['input_channel_schema_matches'].all()
    ),
    'ttp_absent_and_tmax_selected': (
        not bool(vascular_subject_df['ttp_channel_available'].any())
        and bool(vascular_subject_df['tmax_channel_available'].all())
    ),
    'cow_and_lvo_prior_fields_available_for_all': bool(
        vascular_subject_df[[
            'cow_soft_union_available',
            'lvo_proximity_available',
            'lvo_present_field_available',
            'vessel_presence_field_available',
        ]].all().all()
    ),
    'spatial_vascular_stream_available_for_all': bool(
        vascular_subject_df['spatial_vascular_stream_available'].all()
    ),
    'graph_sidecars_contain_no_target_keys': not bool(
        vascular_subject_df['sidecar_contains_forbidden_target_key'].any()
    ),
    'all_atlas_enabled_subjects_have_territory_cta_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_cta_support'
        ].gt(0).all()
    ),
    'all_atlas_enabled_subjects_have_territory_tmax_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_tmax_support'
        ].gt(0).all()
    ),
    'vessel_cta_schema_present': len(vessel_cta_indices) == 2,
    'vessel_tmax_schema_present': len(vessel_tmax_indices) == 1,
    'missing_vascular_graph_inputs_have_validity_masks': bool(
        (
            (~vascular_subject_df['atlas_enabled'])
            | vascular_subject_df['territory_graph_available']
        ).all()
    ),
}

all_checks = {
    **upstream_checks,
    **clinical_checks,
    **vascular_checks,
    'clinical_token_dimension_is_44': clinical_token_input_dim == 44,
    'baseline_architecture_preserved_as_reference': True,
    'new_fusion_paths_planned_zero_initialized': True,
    'direct_graph_to_logit_residual_remains_disabled': True,
    'expert_routing_remains_disabled': True,
    'target_arrays_not_read': True,
    'locked_validation_and_test_inputs_used_for_availability_only': True,
    'study_model_not_constructed': True,
    'study_trainable_parameters_are_zero': True,
    'model_training_remains_locked': True,
}
audit_passed = bool(all(all_checks.values()))

architecture_contract['architecture_implementation_authorized'] = audit_passed
atomic_write_json(architecture_contract, ARCHITECTURE_CONTRACT_PATH)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'audit_version': AUDIT_VERSION,
    'feature_availability_audit_passed': audit_passed,
    'checks': all_checks,
    'manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'manifest_sha256': manifest_sha256,
    'graph_schema_path': str(GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': schema_sha256,
    'baseline_preflight_gate_path': str(BASELINE_PREFLIGHT_GATE_PATH),
    'clinical_feature_audit_path': str(CLINICAL_FEATURE_AUDIT_PATH),
    'clinical_exclusion_audit_path': str(CLINICAL_EXCLUSION_AUDIT_PATH),
    'vascular_feature_audit_path': str(VASCULAR_AUDIT_PATH),
    'subject_audit_path': str(SUBJECT_AUDIT_PATH),
    'architecture_contract_path': str(ARCHITECTURE_CONTRACT_PATH),
    'selected_clinical_scalar_features': clinical_feature_count,
    'clinical_token_input_dimension': clinical_token_input_dim,
    'nihss_subscores_available': False,
    'aggregate_admission_nihss_used': True,
    'ttp_available': False,
    'tmax_used_for_occlusion_source': True,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'clinical_scaler_fitted': False,
    'study_model_constructed': False,
    'study_trainable_parameters': 0,
    'architecture_implementation_authorized': audit_passed,
    'model_training_authorized': False,
    'next_block': (
        'construct and smoke-test VascuFateNet-v2 with exact parameter, '
        'neutrality, ablation, gradient, and L4-memory checks'
    ),
}
atomic_write_json(gate, AUDIT_GATE_PATH)

print('\nPretreatment clinical feature availability:')
display(clinical_feature_df)
print('\nExplicitly excluded clinical fields:')
display(clinical_exclusion_df)
print('\nVascular availability by split:')
display(vascular_audit_df)
print('\nFeature-availability checks:')
display(pd.DataFrame({
    'check': list(all_checks.keys()),
    'passed': list(all_checks.values()),
}))

if not audit_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT FAILED. '
        'Do not construct or train the revised model. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT PASSED')
print('=' * 80)
print(f'Subjects audited:                    {len(subject_audit_df)}')
print(f'Pretreatment clinical scalars:       {clinical_feature_count}')
print(f'Clinical missingness indicators:     {clinical_feature_count}')
print(f'Clinical-token input dimension:      {clinical_token_input_dim}')
print('Admission NIHSS:                     aggregate score available')
print('NIHSS subscores:                     unavailable; not used')
print('Occlusion timing signal:             Tmax (TTP unavailable)')
print('LVO source:                          validity-masked baseline annotation')
print('Outcome workbooks read:              False')
print('Target arrays read:                  False')
print('Clinical scaler fitted:              False')
print('Study trainable parameters:          0 (availability audit only)')
print(f'Baseline parameters preserved:       {EXPECTED_BASELINE_PARAMETERS:,}')
print(f'Architecture implementation allowed: {audit_passed}')
print('Model training authorized:           False')
print(f'Architecture contract:               {ARCHITECTURE_CONTRACT_PATH}')
print(f'Availability gate:                   {AUDIT_GATE_PATH}')
print(
    '\nNext: construct and smoke-test VascuFateNet-v2 with exact parameter, '
    'neutrality, ablation, gradient, and L4-memory checks.'
)


In [ ]:
import sys
import os
import re
import json
import math
import hashlib
import subprocess
from datetime import datetime, time, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
ensure_package('openpyxl')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
RAW_DATA_ROOT = PROJECT_ROOT / 'ISLES24 dataset'
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
REVISED_MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AUDIT_VERSION = 'v1_feature_availability_and_test_time_contract'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_BASELINE_PARAMETERS = 15_052_628
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
    / 'graph_schema_verified_pickle_free.npz'
)
BASELINE_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
BASELINE_PREFLIGHT_GATE_PATH = (
    BASELINE_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / REVISED_MODEL_VERSION
    / AUDIT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CLINICAL_FEATURE_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_availability.csv'
CLINICAL_EXCLUSION_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_exclusions.csv'
SUBJECT_AUDIT_PATH = OUTPUT_ROOT / 'subject_input_availability.csv'
VASCULAR_AUDIT_PATH = OUTPUT_ROOT / 'vascular_feature_availability.csv'
ARCHITECTURE_CONTRACT_PATH = OUTPUT_ROOT / 'architecture_input_contract.json'
AUDIT_GATE_PATH = OUTPUT_ROOT / 'feature_availability_gate.json'

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
VASCULAR_IMAGE_CHANNELS = ['cta', 'tmax', 'perfusion_valid']
VASCULAR_PRIOR_ARRAYS = [
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_presence',
]

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    output = []
    for value in np.asarray(array).reshape(-1):
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def normalize_label(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).strip().lower())


def is_missing(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip().lower() in {
        '', 'na', 'n/a', 'n.a.', 'nan', 'none', 'null', 'missing',
        'unknown', 'not available', 'not applicable', '-', '--',
    }:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def parse_binary(value):
    if is_missing(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        numeric = float(value)
        if numeric in {0.0, 1.0}:
            return numeric
        raise ValueError('binary value is outside 0/1')
    text = str(value).strip().lower()
    mapping = {
        '0': 0.0, 'no': 0.0, 'false': 0.0, 'n': 0.0,
        '1': 1.0, 'yes': 1.0, 'true': 1.0, 'y': 1.0,
    }
    if text in mapping:
        return mapping[text]
    try:
        numeric = float(text)
    except ValueError as error:
        raise ValueError('unrecognized binary value') from error
    if numeric not in {0.0, 1.0}:
        raise ValueError('binary value is outside 0/1')
    return numeric


def parse_sex(value):
    if is_missing(value):
        return None
    text = str(value).strip().lower()
    mapping = {'f': 1.0, 'female': 1.0, 'm': 0.0, 'male': 0.0}
    if text not in mapping:
        raise ValueError('unrecognized sex value')
    return mapping[text]


def parse_numeric(value):
    if is_missing(value):
        return None
    numeric = float(value)
    if not math.isfinite(numeric):
        raise ValueError('non-finite numeric value')
    return numeric


def parse_duration_hours(value):
    if is_missing(value):
        return None
    if isinstance(value, pd.Timedelta):
        hours = value.total_seconds() / 3600.0
    elif isinstance(value, time):
        hours = (
            value.hour + value.minute / 60.0 + value.second / 3600.0
            + value.microsecond / 3.6e9
        )
    elif isinstance(value, (int, float, np.integer, np.floating)):
        
        hours = float(value) * 24.0
    else:
        parsed = pd.to_timedelta(str(value).strip())
        hours = parsed.total_seconds() / 3600.0
    if not math.isfinite(hours) or hours < 0:
        raise ValueError('invalid duration')
    return hours


PARSER_BY_NAME = {
    'binary': parse_binary,
    'sex': parse_sex,
    'numeric': parse_numeric,
    'duration_hours': parse_duration_hours,
}


def read_baseline_table(path):
    suffix = Path(path).suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.tsv', '.txt'}:
        return pd.read_csv(path, sep='\t')
    raise RuntimeError(f'Unsupported baseline table format: {path}')



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'audit_version': AUDIT_VERSION,
        'feature_availability_audit_passed': False,
        'status': 'running_or_incomplete',
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
        'clinical_scaler_fitted': False,
        'study_model_constructed': False,
        'study_trainable_parameters': 0,
        'architecture_implementation_authorized': False,
        'model_training_authorized': False,
    },
    AUDIT_GATE_PATH,
)






required_paths = [
    PROJECT_ROOT,
    RAW_DATA_ROOT,
    TARGET_POLICY_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH,
    BASELINE_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 vascular-fusion audit prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)
schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
baseline_gate = json.loads(
    BASELINE_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)

manifest_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        'Target-policy manifest is missing columns: '
        + ', '.join(missing_manifest_columns)
    )
manifest_df['subject_id'] = (
    manifest_df['subject_id'].astype(str).str.strip().str.lower()
)
manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    schema_arrays = {name: np.asarray(schema[name]) for name in schema.files}
node_feature_names = decode_strings(schema_arrays['node_feature_names'])
vessel_feature_names = decode_strings(schema_arrays['vessel_feature_names'])
case_feature_names = decode_strings(schema_arrays['case_feature_names'])

upstream_checks = {
    'baseline_preflight_passed': baseline_gate.get('preflight_passed') is True,
    'baseline_data_contract_verified': (
        baseline_gate.get('baseline_loader_loss_sampler_contract_verified')
        is True
    ),
    'baseline_training_remained_locked': (
        baseline_gate.get('fold0_pilot_training_authorized') is False
        and baseline_gate.get('full_fivefold_training_authorized') is False
    ),
    'target_policy_manifest_hash_matches': (
        manifest_sha256 == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        schema_sha256 == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream contract failed: ' + ', '.join(failed))

print('Trial8 revised-architecture upstream contract passed.')
print(f'Manifest subjects:                   {len(manifest_df)}')
print(f'Split counts:                        {EXPECTED_SPLITS}')
print(f'Frozen manifest SHA-256:             {manifest_sha256}')
print(f'Pickle-free schema SHA-256:          {schema_sha256}')
print('Baseline training authorized:        False')
print('Target arrays read:                  False')
print('Outcome workbooks read:              False')
print('Study trainable parameters:          0 (availability audit only)')









CLINICAL_FEATURE_SPECS = [
    ('sex_female', ['Sex'], 'sex'),
    ('age_years', ['Age'], 'numeric'),
    ('atrial_fibrillation', ['Atrial fibrillation'], 'binary'),
    ('hypertension', ['Hypertension'], 'binary'),
    ('diabetes', ['Diabetes'], 'binary'),
    ('hyperlipidemia', ['Hyperlipidemia'], 'binary'),
    ('anticoagulation', ['Anticoagulation'], 'binary'),
    ('lipid_lowering_drugs', ['Lipid lowering drugs', 'Statins'], 'binary'),
    ('platelet_aggregation_inhibitors', ['PAIs'], 'binary'),
    ('glucose_mg_dl', ['Glucose'], 'numeric'),
    ('leucocytes_g_l', ['Leucocytes'], 'numeric'),
    ('crp_mg_dl', ['CRP'], 'numeric'),
    ('inr', ['INR'], 'numeric'),
    ('wake_up_stroke', ['Wake-up'], 'binary'),
    ('in_house_stroke', ['In-House', 'In-house'], 'binary'),
    ('external_referral', ['Referral'], 'binary'),
    ('onset_to_door_hours', ['Onset to door'], 'duration_hours'),
    ('alert_to_door_hours', ['Alert to door'], 'duration_hours'),
    ('nihss_at_admission', ['NIHSS at admission'], 'numeric'),
    ('mrs_at_admission', ['mRS at admission', 'MRS at admission'], 'numeric'),
    ('mrs_premorbid', ['mRS premorbid', 'MRS premorbid'], 'numeric'),
    ('door_to_imaging_hours', ['Door to imaging'], 'duration_hours'),
]

EXCLUDED_CLINICAL_SPECS = [
    ('Center', 'site_identifier_and_domain_shortcut'),
    ('Identifier', 'identifier'),
    ('Door to groin', 'post_imaging_treatment_timing'),
    ('Door to first series', 'post_imaging_treatment_timing'),
    ('Time of intervention', 'post_imaging_treatment_timing'),
    ('Door to recanalization', 'post_imaging_treatment_timing'),
    ('TICI postinterventional', 'post_treatment_outcome'),
    ('mTici postinterventional', 'post_treatment_outcome'),
    ('NIHSS 24h', 'post_treatment_outcome'),
    ('mRS 24h', 'post_treatment_outcome'),
    ('NIHSS discharge', 'post_treatment_outcome'),
    ('mRS discharge', 'post_treatment_outcome'),
    ('MRS discharge', 'post_treatment_outcome'),
    ('mRS 3 months', 'post_treatment_outcome'),
    ('MRS 3 months', 'post_treatment_outcome'),
]

selected_normalized_names = {
    normalize_label(alias)
    for _, aliases, _ in CLINICAL_FEATURE_SPECS
    for alias in aliases
}
excluded_normalized_names = {
    normalize_label(name) for name, _ in EXCLUDED_CLINICAL_SPECS
}
if selected_normalized_names & excluded_normalized_names:
    raise RuntimeError('Selected and excluded clinical fields overlap.')


def resolve_baseline_row(frame, subject_id):
    
    observed_rows = len(frame)
    if observed_rows < 1:
        raise RuntimeError(f'{subject_id}: baseline workbook has no data row.')
    if observed_rows == 1:
        return frame.iloc[0], {
            'baseline_rows_observed': 1,
            'baseline_unique_rows': 1,
            'row_selection_method': 'single_row',
            'selected_source_row_index': str(frame.index[0]),
            'duplicate_rows_collapsed': 0,
        }

    
    
    
    unique_frame = frame.drop_duplicates(keep='first')
    if len(unique_frame) == 1:
        return unique_frame.iloc[0], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': 1,
            'row_selection_method': 'exact_duplicate_rows_collapsed',
            'selected_source_row_index': str(unique_frame.index[0]),
            'duplicate_rows_collapsed': observed_rows - 1,
        }

    normalized_to_original = {
        normalize_label(column): str(column) for column in unique_frame.columns
    }

    
    
    
    
    identifier_columns = [
        normalized_to_original[name]
        for name in ['identifier', 'subjectid', 'participantid', 'patientid']
        if name in normalized_to_original
    ]
    target_digits = ''.join(re.findall(r'\d+', subject_id)).lstrip('0') or '0'
    matched_indices = []
    for row_index, candidate in unique_frame.iterrows():
        for column in identifier_columns:
            value_digits = ''.join(re.findall(r'\d+', str(candidate[column])))
            value_digits = value_digits.lstrip('0') or '0'
            if value_digits == target_digits:
                matched_indices.append(row_index)
                break
    if len(matched_indices) == 1:
        selected_index = matched_indices[0]
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_subject_identifier_match',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }

    
    
    
    
    profiles = []
    for row_index, candidate in unique_frame.iterrows():
        valid_count = 0
        parse_errors = 0
        for _, aliases, parser_name in CLINICAL_FEATURE_SPECS:
            source_column = next(
                (
                    normalized_to_original[normalize_label(alias)]
                    for alias in aliases
                    if normalize_label(alias) in normalized_to_original
                ),
                None,
            )
            if source_column is None:
                continue
            try:
                parsed = PARSER_BY_NAME[parser_name](candidate[source_column])
            except Exception:
                parse_errors += 1
                continue
            if parsed is not None:
                valid_count += 1
        profiles.append({
            'row_index': row_index,
            'valid_selected_fields': valid_count,
            'parse_errors': parse_errors,
        })

    ordered = sorted(
        profiles,
        key=lambda item: (item['parse_errors'], -item['valid_selected_fields']),
    )
    best = ordered[0]
    runner_up = ordered[1]
    if (
        best['parse_errors'] == 0
        and best['valid_selected_fields']
        >= runner_up['valid_selected_fields'] + 3
    ):
        selected_index = best['row_index']
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_parse_complete_patient_row',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }

    profile_text = '; '.join(
        f"row={item['row_index']},valid={item['valid_selected_fields']},"
        f"errors={item['parse_errors']}"
        for item in profiles
    )
    raise RuntimeError(
        f'{subject_id}: {observed_rows} baseline rows remain ambiguous after '
        f'exact-deduplication ({len(unique_frame)} unique). Profiles: '
        f'{profile_text}. No row was selected.'
    )

clinical_exclusion_df = pd.DataFrame([
    {'source_field': name, 'exclusion_reason': reason, 'model_input': False}
    for name, reason in EXCLUDED_CLINICAL_SPECS
])
atomic_write_csv(clinical_exclusion_df, CLINICAL_EXCLUSION_AUDIT_PATH)






allowed_table_suffixes = {'.xlsx', '.xls', '.csv', '.tsv'}
baseline_candidates = sorted(
    path for path in RAW_DATA_ROOT.rglob('*demographic_baseline*')
    if path.is_file() and path.suffix.lower() in allowed_table_suffixes
)
manifest_subjects = set(manifest_df['subject_id'].astype(str))
baseline_by_subject = {}
unmapped_baseline_files = []
for path in baseline_candidates:
    match = re.search(r'(sub-stroke\d+)', str(path), flags=re.IGNORECASE)
    if match is None:
        unmapped_baseline_files.append(str(path))
        continue
    subject_id = match.group(1).lower()
    if subject_id not in manifest_subjects:
        continue
    if subject_id in baseline_by_subject:
        raise RuntimeError(
            f'{subject_id}: multiple baseline clinical workbooks found.'
        )
    baseline_by_subject[subject_id] = path

clinical_feature_records = {
    name: {
        'model_feature': name,
        'aliases': aliases,
        'parser': parser_name,
        'subjects_total': 0,
        'source_columns_resolved': 0,
        'subjects_present': 0,
        'subjects_missing': 0,
        'parse_errors': 0,
        'train_present': 0,
        'val_present': 0,
        'test_present': 0,
        'resolved_source_columns': set(),
    }
    for name, aliases, parser_name in CLINICAL_FEATURE_SPECS
}

subject_clinical_rows = []
clinical_schema_sets = []
nihss_subscore_columns = set()
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id']).lower()
    split = str(manifest_row['dev_split'])
    baseline_path = baseline_by_subject.get(subject_id)
    if baseline_path is None:
        subject_clinical_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'baseline_clinical_path': '',
            'baseline_clinical_sha256': '',
            'baseline_rows_observed': 0,
            'baseline_unique_rows': 0,
            'row_selection_method': '',
            'selected_source_row_index': '',
            'duplicate_rows_collapsed': 0,
            'baseline_columns': 0,
            'selected_features_present': 0,
            'selected_features_missing': len(CLINICAL_FEATURE_SPECS),
            'clinical_parse_errors': 0,
            'clinical_parse_error_features': '',
            'aggregate_admission_nihss_available': False,
            'nihss_subscore_count': 0,
        })
        continue

    frame = read_baseline_table(baseline_path)
    frame = frame.dropna(axis=0, how='all').dropna(axis=1, how='all')
    row, row_resolution = resolve_baseline_row(frame, subject_id)
    normalized_to_original = {
        normalize_label(column): str(column) for column in frame.columns
    }
    clinical_schema_sets.append(tuple(sorted(normalized_to_original)))
    subject_subscores = [
        str(column)
        for column in frame.columns
        if 'nihss' in normalize_label(column)
        and normalize_label(column) != normalize_label('NIHSS at admission')
    ]
    nihss_subscore_columns.update(subject_subscores)

    present_count = 0
    missing_count = 0
    parse_error_count = 0
    parse_error_features = []
    aggregate_nihss_available = False
    for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
        record = clinical_feature_records[model_name]
        record['subjects_total'] += 1
        source_column = next(
            (
                normalized_to_original[normalize_label(alias)]
                for alias in aliases
                if normalize_label(alias) in normalized_to_original
            ),
            None,
        )
        if source_column is None:
            record['subjects_missing'] += 1
            missing_count += 1
            continue
        record['source_columns_resolved'] += 1
        record['resolved_source_columns'].add(source_column)
        value = row[source_column]
        try:
            parsed = PARSER_BY_NAME[parser_name](value)
        except Exception:
            parsed = None
            record['parse_errors'] += 1
            parse_error_count += 1
            parse_error_features.append(model_name)
        if parsed is None:
            record['subjects_missing'] += 1
            missing_count += 1
        else:
            record['subjects_present'] += 1
            record[f'{split}_present'] += 1
            present_count += 1
            if model_name == 'nihss_at_admission':
                aggregate_nihss_available = True

    subject_clinical_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'baseline_clinical_path': str(baseline_path),
        'baseline_clinical_sha256': file_sha256(baseline_path),
        **row_resolution,
        'baseline_columns': len(frame.columns),
        'selected_features_present': present_count,
        'selected_features_missing': missing_count,
        'clinical_parse_errors': parse_error_count,
        'clinical_parse_error_features': ' | '.join(parse_error_features),
        'aggregate_admission_nihss_available': aggregate_nihss_available,
        'nihss_subscore_count': len(subject_subscores),
    })

clinical_rows = []
for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
    record = clinical_feature_records[model_name]
    clinical_rows.append({
        'model_feature': model_name,
        'source_aliases': ' | '.join(aliases),
        'resolved_source_columns': ' | '.join(
            sorted(record['resolved_source_columns'])
        ),
        'parser': parser_name,
        'subjects_total': record['subjects_total'],
        'source_columns_resolved': record['source_columns_resolved'],
        'subjects_present': record['subjects_present'],
        'subjects_missing': record['subjects_missing'],
        'missing_fraction': (
            record['subjects_missing'] / max(record['subjects_total'], 1)
        ),
        'parse_errors': record['parse_errors'],
        'train_present': record['train_present'],
        'val_present': record['val_present'],
        'test_present': record['test_present'],
        'paired_missingness_indicator': True,
        'scaler_policy': (
            'fit_on_development_fold_train_only_then_freeze'
            if parser_name in {'numeric', 'duration_hours'}
            else 'not_scaled_binary'
        ),
    })

clinical_feature_df = pd.DataFrame(clinical_rows)
clinical_subject_df = pd.DataFrame(subject_clinical_rows)
raw_baseline_schema_variant_count = len(set(clinical_schema_sets))
atomic_write_csv(clinical_feature_df, CLINICAL_FEATURE_AUDIT_PATH)






node_cta_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('cta_')
]
node_tmax_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('tmax_')
]
vessel_cta_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name in {'cta_mean', 'cta_p90'}
]
vessel_tmax_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name == 'tmax_mean'
]

vascular_rows = []
subject_vascular_rows = []
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id'])
    split = str(manifest_row['dev_split'])
    npz_path = Path(str(manifest_row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(manifest_row['trial8_graph_sidecar_path']))
    if not npz_path.exists():
        raise FileNotFoundError(f'{subject_id}: NPZ is missing: {npz_path}')
    if not sidecar_path.exists():
        raise FileNotFoundError(
            f'{subject_id}: graph sidecar is missing: {sidecar_path}'
        )
    if file_sha256(sidecar_path) != str(
        manifest_row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')

    with np.load(npz_path, allow_pickle=False) as data:
        npz_keys = set(data.files)
        missing_safe_keys = sorted(
            {'input_channels', *VASCULAR_PRIOR_ARRAYS} - npz_keys
        )
        input_channels = (
            decode_strings(data['input_channels'])
            if 'input_channels' in npz_keys
            else []
        )
        lvo_present = (
            bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            if 'lvo_present' in npz_keys
            else False
        )
        vessel_presence_count = (
            int((np.asarray(data['vessel_presence']) > 0).sum())
            if 'vessel_presence' in npz_keys
            else 0
        )

    with np.load(sidecar_path, allow_pickle=False) as data:
        sidecar_keys = set(data.files)
        forbidden_in_sidecar = sorted(FORBIDDEN_TARGET_KEYS & sidecar_keys)
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0

    atlas_enabled = as_bool(manifest_row['trial8_atlas_enabled'])
    valid_territory_nodes = int(node_valid.sum())
    valid_vessel_nodes = int(vessel_valid.sum())
    territory_cta_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_cta_indices]).sum()
    )
    territory_tmax_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_tmax_indices]).sum()
    )
    vessel_cta_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_cta_indices]).sum()
    )
    vessel_tmax_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_tmax_indices]).sum()
    )
    subject_vascular_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'input_channel_schema_matches': input_channels == EXPECTED_INPUT_CHANNELS,
        'cta_channel_available': 'cta' in input_channels,
        'tmax_channel_available': 'tmax' in input_channels,
        'ttp_channel_available': 'ttp' in input_channels,
        'perfusion_valid_channel_available': 'perfusion_valid' in input_channels,
        'cow_soft_union_available': 'cow_soft_union' in npz_keys,
        'lvo_proximity_available': 'lvo_proximity' in npz_keys,
        'lvo_present_field_available': 'lvo_present' in npz_keys,
        'lvo_present': lvo_present,
        'vessel_presence_field_available': 'vessel_presence' in npz_keys,
        'raw_vessel_presence_count': vessel_presence_count,
        'missing_safe_npz_keys': ' | '.join(missing_safe_keys),
        'sidecar_contains_forbidden_target_key': bool(forbidden_in_sidecar),
        'forbidden_sidecar_keys': ' | '.join(forbidden_in_sidecar),
        'atlas_enabled': atlas_enabled,
        'valid_territory_nodes': valid_territory_nodes,
        'valid_vessel_nodes': valid_vessel_nodes,
        'territory_cta_support': territory_cta_support,
        'territory_tmax_support': territory_tmax_support,
        'vessel_cta_support': vessel_cta_support,
        'vessel_tmax_support': vessel_tmax_support,
        'spatial_vascular_stream_available': (
            input_channels == EXPECTED_INPUT_CHANNELS
            and not missing_safe_keys
        ),
        'territory_graph_available': (
            atlas_enabled and valid_territory_nodes > 0
        ),
        'vessel_graph_available': valid_vessel_nodes > 0,
        'occlusion_source_valid': lvo_present,
    })

vascular_subject_df = pd.DataFrame(subject_vascular_rows)
for split, group in vascular_subject_df.groupby('dev_split', sort=True):
    vascular_rows.append({
        'dev_split': split,
        'subjects': len(group),
        'spatial_vascular_stream_available': int(
            group['spatial_vascular_stream_available'].sum()
        ),
        'territory_graph_available': int(
            group['territory_graph_available'].sum()
        ),
        'vessel_graph_available': int(group['vessel_graph_available'].sum()),
        'occlusion_source_valid': int(group['occlusion_source_valid'].sum()),
        'minimum_valid_territory_nodes': int(
            group['valid_territory_nodes'].min()
        ),
        'minimum_valid_vessel_nodes': int(group['valid_vessel_nodes'].min()),
        'mean_valid_vessel_nodes': float(group['valid_vessel_nodes'].mean()),
    })
vascular_audit_df = pd.DataFrame(vascular_rows)
atomic_write_csv(vascular_audit_df, VASCULAR_AUDIT_PATH)

subject_audit_df = clinical_subject_df.merge(
    vascular_subject_df,
    on=['subject_id', 'dev_split'],
    how='outer',
    validate='one_to_one',
)
atomic_write_csv(subject_audit_df, SUBJECT_AUDIT_PATH)






clinical_feature_count = len(CLINICAL_FEATURE_SPECS)
clinical_token_input_dim = clinical_feature_count * 2

architecture_contract = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'architecture_version': REVISED_MODEL_VERSION,
    'status': 'input_contract_only_model_not_constructed',
    'preserved_reference_path': {
        'image_backbone': '15-channel compact residual 3D U-Net',
        'graph_encoder': 'heterogeneous relation-aware GATv2',
        'graph_modulation': 'bounded zero-init multiscale FiLM',
        'reference_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    },
    'asymmetric_vascular_stream': {
        'spatial_inputs': VASCULAR_IMAGE_CHANNELS + [
            'cow_soft_union', 'lvo_proximity'
        ],
        'planned_channels': [8, 16, 32, 48],
        'planned_fusion_scales': ['quarter', 'eighth', 'sixteenth', 'bottleneck'],
        'fusion': 'confidence_gated_zero_initialized_residual_adapters',
        'confidence_inputs': [
            'perfusion_valid',
            'cow_prior_valid',
            'registration_confidence',
            'lvo_present',
        ],
        'purpose': (
            'emphasize CTA/perfusion/arterial structure without duplicating '
            'the full image backbone'
        ),
    },
    'occlusion_source': {
        'representation': 'validity_masked_source_token_or_graph_node',
        'inputs': [
            'lvo_present',
            'lvo_proximity',
            'vessel_cta_mean',
            'vessel_cta_p90',
            'vessel_tmax_mean',
            'vessel_minimum_lvo_distance_mm',
            'vessel_lvo_overlap_fraction',
        ],
        'ttp_used': False,
        'ttp_reason': 'TTP is not present in the frozen 15-channel input schema',
        'tmax_used': True,
        'invalid_state': 'zero token plus explicit validity bit',
        'external_deployment_dependency': (
            'requires the provided baseline LVO annotation or a separately '
            'validated CTA-derived occlusion detector'
        ),
    },
    'clinical_token': {
        'source': 'ses-01 demographic_baseline only',
        'selected_scalar_features': [
            name for name, _, _ in CLINICAL_FEATURE_SPECS
        ],
        'selected_scalar_count': clinical_feature_count,
        'missingness_indicator_count': clinical_feature_count,
        'input_dimension': clinical_token_input_dim,
        'planned_embedding_dimension': 32,
        'nihss_policy': 'aggregate admission NIHSS only; no subscores available',
        'missing_data_policy': (
            'fold-train median for continuous features, neutral zero for '
            'binary features, and one explicit missingness bit per scalar'
        ),
        'scaler_policy': 'fit inside each development fold train partition only',
        'raw_baseline_schema_variant_count': raw_baseline_schema_variant_count,
        'raw_schema_variation_policy': (
            'informational only; every selected source field must still resolve '
            'for every subject'
        ),
        'excluded_fields': clinical_exclusion_df.to_dict('records'),
    },
    'anti_leakage': {
        'outcome_workbooks_read': False,
        'target_arrays_read': False,
        'center_used': False,
        'post_imaging_treatment_times_used': False,
        'post_treatment_outcomes_used': False,
        'clinical_scaler_fitted_in_this_audit': False,
    },
    'regularization_contract': {
        'zero_initialized_new_fusion_paths': True,
        'independent_vascular_branch_dropout': True,
        'clinical_token_dropout': True,
        'atlas_and_occlusion_source_dropout': True,
        'missing_inputs_use_explicit_validity_masks': True,
        'direct_graph_to_logit_residual': False,
        'expert_routing': False,
    },
    'parameter_budget': {
        'baseline_parameters': EXPECTED_BASELINE_PARAMETERS,
        'planned_total_minimum': 15_300_000,
        'planned_total_maximum': 15_650_000,
        'exact_count_required_in_next_model_preflight': True,
    },
    'architecture_implementation_authorized': True,
    'model_training_authorized': False,
}

clinical_checks = {
    'exactly_one_baseline_workbook_per_subject': (
        len(baseline_by_subject) == EXPECTED_SUBJECTS
        and clinical_subject_df['baseline_clinical_path'].astype(str).str.len().gt(0).all()
    ),
    'all_baseline_workbooks_resolve_to_one_unambiguous_patient_row': bool(
        clinical_subject_df['row_selection_method'].fillna('').str.len().gt(0).all()
    ),
    
    
    'all_selected_clinical_columns_resolve_for_all_subjects': bool(
        clinical_feature_df['source_columns_resolved'].eq(EXPECTED_SUBJECTS).all()
    ),
    'all_selected_features_have_train_support': bool(
        clinical_feature_df['train_present'].gt(0).all()
    ),
    'all_nonmissing_selected_values_parse': bool(
        clinical_feature_df['parse_errors'].eq(0).all()
    ),
    'aggregate_admission_nihss_available': bool(
        clinical_feature_df.loc[
            clinical_feature_df['model_feature'] == 'nihss_at_admission',
            'train_present',
        ].gt(0).all()
    ),
    'nihss_subscores_are_not_available_or_claimed': (
        len(nihss_subscore_columns) == 0
        and clinical_subject_df['nihss_subscore_count'].eq(0).all()
    ),
    'center_is_excluded': 'center' not in {
        name for name, _, _ in CLINICAL_FEATURE_SPECS
    },
    'post_imaging_treatment_fields_are_excluded': all(
        normalize_label(name) not in selected_normalized_names
        for name in [
            'Door to groin', 'Door to first series',
            'Time of intervention', 'Door to recanalization',
        ]
    ),
    'outcome_workbooks_not_read': True,
    'clinical_scaler_not_fitted': True,
}

vascular_checks = {
    'cta_tmax_and_perfusion_valid_channels_available_for_all': bool(
        vascular_subject_df[[
            'cta_channel_available',
            'tmax_channel_available',
            'perfusion_valid_channel_available',
        ]].all().all()
    ),
    'input_channel_schema_matches_for_all': bool(
        vascular_subject_df['input_channel_schema_matches'].all()
    ),
    'ttp_absent_and_tmax_selected': (
        not bool(vascular_subject_df['ttp_channel_available'].any())
        and bool(vascular_subject_df['tmax_channel_available'].all())
    ),
    'cow_and_lvo_prior_fields_available_for_all': bool(
        vascular_subject_df[[
            'cow_soft_union_available',
            'lvo_proximity_available',
            'lvo_present_field_available',
            'vessel_presence_field_available',
        ]].all().all()
    ),
    'spatial_vascular_stream_available_for_all': bool(
        vascular_subject_df['spatial_vascular_stream_available'].all()
    ),
    'graph_sidecars_contain_no_target_keys': not bool(
        vascular_subject_df['sidecar_contains_forbidden_target_key'].any()
    ),
    'all_atlas_enabled_subjects_have_territory_cta_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_cta_support'
        ].gt(0).all()
    ),
    'all_atlas_enabled_subjects_have_territory_tmax_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_tmax_support'
        ].gt(0).all()
    ),
    'vessel_cta_schema_present': len(vessel_cta_indices) == 2,
    'vessel_tmax_schema_present': len(vessel_tmax_indices) == 1,
    'missing_vascular_graph_inputs_have_validity_masks': bool(
        (
            (~vascular_subject_df['atlas_enabled'])
            | vascular_subject_df['territory_graph_available']
        ).all()
    ),
}

all_checks = {
    **upstream_checks,
    **clinical_checks,
    **vascular_checks,
    'clinical_token_dimension_is_44': clinical_token_input_dim == 44,
    'baseline_architecture_preserved_as_reference': True,
    'new_fusion_paths_planned_zero_initialized': True,
    'direct_graph_to_logit_residual_remains_disabled': True,
    'expert_routing_remains_disabled': True,
    'target_arrays_not_read': True,
    'locked_validation_and_test_inputs_used_for_availability_only': True,
    'study_model_not_constructed': True,
    'study_trainable_parameters_are_zero': True,
    'model_training_remains_locked': True,
}
audit_passed = bool(all(all_checks.values()))

architecture_contract['architecture_implementation_authorized'] = audit_passed
atomic_write_json(architecture_contract, ARCHITECTURE_CONTRACT_PATH)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'audit_version': AUDIT_VERSION,
    'feature_availability_audit_passed': audit_passed,
    'checks': all_checks,
    'manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'manifest_sha256': manifest_sha256,
    'graph_schema_path': str(GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': schema_sha256,
    'baseline_preflight_gate_path': str(BASELINE_PREFLIGHT_GATE_PATH),
    'clinical_feature_audit_path': str(CLINICAL_FEATURE_AUDIT_PATH),
    'clinical_exclusion_audit_path': str(CLINICAL_EXCLUSION_AUDIT_PATH),
    'vascular_feature_audit_path': str(VASCULAR_AUDIT_PATH),
    'subject_audit_path': str(SUBJECT_AUDIT_PATH),
    'architecture_contract_path': str(ARCHITECTURE_CONTRACT_PATH),
    'selected_clinical_scalar_features': clinical_feature_count,
    'clinical_token_input_dimension': clinical_token_input_dim,
    'raw_baseline_schema_variant_count': raw_baseline_schema_variant_count,
    'nihss_subscores_available': False,
    'aggregate_admission_nihss_used': True,
    'ttp_available': False,
    'tmax_used_for_occlusion_source': True,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'clinical_scaler_fitted': False,
    'study_model_constructed': False,
    'study_trainable_parameters': 0,
    'architecture_implementation_authorized': audit_passed,
    'model_training_authorized': False,
    'next_block': (
        'construct and smoke-test VascuFateNet-v2 with exact parameter, '
        'neutrality, ablation, gradient, and L4-memory checks'
    ),
}
atomic_write_json(gate, AUDIT_GATE_PATH)

print('\nPretreatment clinical feature availability:')
display(clinical_feature_df)
print('\nExplicitly excluded clinical fields:')
display(clinical_exclusion_df)
print('\nVascular availability by split:')
display(vascular_audit_df)
print('\nFeature-availability checks:')
display(pd.DataFrame({
    'check': list(all_checks.keys()),
    'passed': list(all_checks.values()),
}))

if not audit_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT FAILED. '
        'Do not construct or train the revised model. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT PASSED')
print('=' * 80)
print(f'Subjects audited:                    {len(subject_audit_df)}')
print(f'Pretreatment clinical scalars:       {clinical_feature_count}')
print(f'Clinical missingness indicators:     {clinical_feature_count}')
print(f'Clinical-token input dimension:      {clinical_token_input_dim}')
print(f'Raw baseline schema variants:        {raw_baseline_schema_variant_count}')
print('Admission NIHSS:                     aggregate score available')
print('NIHSS subscores:                     unavailable; not used')
print('Occlusion timing signal:             Tmax (TTP unavailable)')
print('LVO source:                          validity-masked baseline annotation')
print('Outcome workbooks read:              False')
print('Target arrays read:                  False')
print('Clinical scaler fitted:              False')
print('Study trainable parameters:          0 (availability audit only)')
print(f'Baseline parameters preserved:       {EXPECTED_BASELINE_PARAMETERS:,}')
print(f'Architecture implementation allowed: {audit_passed}')
print('Model training authorized:           False')
print(f'Architecture contract:               {ARCHITECTURE_CONTRACT_PATH}')
print(f'Availability gate:                   {AUDIT_GATE_PATH}')
print(
    '\nNext: construct and smoke-test VascuFateNet-v2 with exact parameter, '
    'neutrality, ablation, gradient, and L4-memory checks.'
)


In [ ]:
import sys
import os
import re
import json
import math
import hashlib
import subprocess
from datetime import datetime, time, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
ensure_package('openpyxl')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
RAW_DATA_ROOT = PROJECT_ROOT / 'ISLES24 dataset'
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
REVISED_MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AUDIT_VERSION = 'v1_feature_availability_and_test_time_contract'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_BASELINE_PARAMETERS = 15_052_628
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
    / 'graph_schema_verified_pickle_free.npz'
)
BASELINE_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
BASELINE_PREFLIGHT_GATE_PATH = (
    BASELINE_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / REVISED_MODEL_VERSION
    / AUDIT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CLINICAL_FEATURE_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_availability.csv'
CLINICAL_EXCLUSION_AUDIT_PATH = OUTPUT_ROOT / 'clinical_feature_exclusions.csv'
SUBJECT_AUDIT_PATH = OUTPUT_ROOT / 'subject_input_availability.csv'
VASCULAR_AUDIT_PATH = OUTPUT_ROOT / 'vascular_feature_availability.csv'
ARCHITECTURE_CONTRACT_PATH = OUTPUT_ROOT / 'architecture_input_contract.json'
AUDIT_GATE_PATH = OUTPUT_ROOT / 'feature_availability_gate.json'

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
VASCULAR_IMAGE_CHANNELS = ['cta', 'tmax', 'perfusion_valid']
VASCULAR_PRIOR_ARRAYS = [
    'cow_soft_union',
    'lvo_proximity',
    'lvo_present',
    'vessel_presence',
]

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    output = []
    for value in np.asarray(array).reshape(-1):
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def normalize_label(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).strip().lower())


def is_missing(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip().lower() in {
        '', 'na', 'n/a', 'n.a.', 'nan', 'none', 'null', 'missing',
        'unknown', 'not available', 'not applicable', '-', '--',
    }:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def parse_binary(value):
    if is_missing(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        numeric = float(value)
        if numeric in {0.0, 1.0}:
            return numeric
        raise ValueError('binary value is outside 0/1')
    text = str(value).strip().lower()
    mapping = {
        '0': 0.0, 'no': 0.0, 'false': 0.0, 'n': 0.0,
        '1': 1.0, 'yes': 1.0, 'true': 1.0, 'y': 1.0,
    }
    if text in mapping:
        return mapping[text]
    try:
        numeric = float(text)
    except ValueError as error:
        raise ValueError('unrecognized binary value') from error
    if numeric not in {0.0, 1.0}:
        raise ValueError('binary value is outside 0/1')
    return numeric


def parse_sex(value):
    if is_missing(value):
        return None
    text = str(value).strip().lower()
    mapping = {'f': 1.0, 'female': 1.0, 'm': 0.0, 'male': 0.0}
    if text not in mapping:
        raise ValueError('unrecognized sex value')
    return mapping[text]


def parse_numeric(value):
    if is_missing(value):
        return None
    numeric = float(value)
    if not math.isfinite(numeric):
        raise ValueError('non-finite numeric value')
    return numeric


def parse_duration_hours(value):
    if is_missing(value):
        return None
    if isinstance(value, pd.Timedelta):
        hours = value.total_seconds() / 3600.0
    elif isinstance(value, time):
        hours = (
            value.hour + value.minute / 60.0 + value.second / 3600.0
            + value.microsecond / 3.6e9
        )
    elif isinstance(value, (int, float, np.integer, np.floating)):
        
        hours = float(value) * 24.0
    else:
        parsed = pd.to_timedelta(str(value).strip())
        hours = parsed.total_seconds() / 3600.0
    if not math.isfinite(hours) or hours < 0:
        raise ValueError('invalid duration')
    return hours


PARSER_BY_NAME = {
    'binary': parse_binary,
    'sex': parse_sex,
    'numeric': parse_numeric,
    'duration_hours': parse_duration_hours,
}


def read_baseline_table(path):
    suffix = Path(path).suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.tsv', '.txt'}:
        return pd.read_csv(path, sep='\t')
    raise RuntimeError(f'Unsupported baseline table format: {path}')



atomic_write_json(
    {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'audit_version': AUDIT_VERSION,
        'feature_availability_audit_passed': False,
        'status': 'running_or_incomplete',
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
        'clinical_scaler_fitted': False,
        'study_model_constructed': False,
        'study_trainable_parameters': 0,
        'architecture_implementation_authorized': False,
        'model_training_authorized': False,
    },
    AUDIT_GATE_PATH,
)






required_paths = [
    PROJECT_ROOT,
    RAW_DATA_ROOT,
    TARGET_POLICY_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH,
    BASELINE_PREFLIGHT_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 vascular-fusion audit prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)
schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
baseline_gate = json.loads(
    BASELINE_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)

manifest_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
if missing_manifest_columns:
    raise RuntimeError(
        'Target-policy manifest is missing columns: '
        + ', '.join(missing_manifest_columns)
    )
manifest_df['subject_id'] = (
    manifest_df['subject_id'].astype(str).str.strip().str.lower()
)
manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    schema_arrays = {name: np.asarray(schema[name]) for name in schema.files}
node_feature_names = decode_strings(schema_arrays['node_feature_names'])
vessel_feature_names = decode_strings(schema_arrays['vessel_feature_names'])
case_feature_names = decode_strings(schema_arrays['case_feature_names'])

upstream_checks = {
    'baseline_preflight_passed': baseline_gate.get('preflight_passed') is True,
    'baseline_data_contract_verified': (
        baseline_gate.get('baseline_loader_loss_sampler_contract_verified')
        is True
    ),
    'baseline_training_remained_locked': (
        baseline_gate.get('fold0_pilot_training_authorized') is False
        and baseline_gate.get('full_fivefold_training_authorized') is False
    ),
    'target_policy_manifest_hash_matches': (
        manifest_sha256 == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        schema_sha256 == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Upstream contract failed: ' + ', '.join(failed))

print('Trial8 revised-architecture upstream contract passed.')
print(f'Manifest subjects:                   {len(manifest_df)}')
print(f'Split counts:                        {EXPECTED_SPLITS}')
print(f'Frozen manifest SHA-256:             {manifest_sha256}')
print(f'Pickle-free schema SHA-256:          {schema_sha256}')
print('Baseline training authorized:        False')
print('Target arrays read:                  False')
print('Outcome workbooks read:              False')
print('Study trainable parameters:          0 (availability audit only)')









CLINICAL_FEATURE_SPECS = [
    ('sex_female', ['Sex'], 'sex'),
    ('age_years', ['Age'], 'numeric'),
    ('atrial_fibrillation', ['Atrial fibrillation'], 'binary'),
    ('hypertension', ['Hypertension'], 'binary'),
    ('diabetes', ['Diabetes'], 'binary'),
    ('hyperlipidemia', ['Hyperlipidemia'], 'binary'),
    ('anticoagulation', ['Anticoagulation'], 'binary'),
    ('lipid_lowering_drugs', ['Lipid lowering drugs', 'Statins'], 'binary'),
    ('platelet_aggregation_inhibitors', ['PAIs'], 'binary'),
    ('glucose_mg_dl', ['Glucose'], 'numeric'),
    ('leucocytes_g_l', ['Leucocytes'], 'numeric'),
    ('crp_mg_dl', ['CRP'], 'numeric'),
    ('inr', ['INR'], 'numeric'),
    ('wake_up_stroke', ['Wake-up'], 'binary'),
    ('in_house_stroke', ['In-House', 'In-house'], 'binary'),
    ('external_referral', ['Referral'], 'binary'),
    ('onset_to_door_hours', ['Onset to door'], 'duration_hours'),
    ('alert_to_door_hours', ['Alert to door'], 'duration_hours'),
    ('nihss_at_admission', ['NIHSS at admission'], 'numeric'),
    ('mrs_at_admission', ['mRS at admission', 'MRS at admission'], 'numeric'),
    ('mrs_premorbid', ['mRS premorbid', 'MRS premorbid'], 'numeric'),
    ('door_to_imaging_hours', ['Door to imaging'], 'duration_hours'),
]

EXCLUDED_CLINICAL_SPECS = [
    ('Center', 'site_identifier_and_domain_shortcut'),
    ('Identifier', 'identifier'),
    ('Door to groin', 'post_imaging_treatment_timing'),
    ('Door to first series', 'post_imaging_treatment_timing'),
    ('Time of intervention', 'post_imaging_treatment_timing'),
    ('Door to recanalization', 'post_imaging_treatment_timing'),
    ('TICI postinterventional', 'post_treatment_outcome'),
    ('mTici postinterventional', 'post_treatment_outcome'),
    ('NIHSS 24h', 'post_treatment_outcome'),
    ('mRS 24h', 'post_treatment_outcome'),
    ('NIHSS discharge', 'post_treatment_outcome'),
    ('mRS discharge', 'post_treatment_outcome'),
    ('MRS discharge', 'post_treatment_outcome'),
    ('mRS 3 months', 'post_treatment_outcome'),
    ('MRS 3 months', 'post_treatment_outcome'),
]

selected_normalized_names = {
    normalize_label(alias)
    for _, aliases, _ in CLINICAL_FEATURE_SPECS
    for alias in aliases
}
excluded_normalized_names = {
    normalize_label(name) for name, _ in EXCLUDED_CLINICAL_SPECS
}
if selected_normalized_names & excluded_normalized_names:
    raise RuntimeError('Selected and excluded clinical fields overlap.')


def resolve_baseline_row(frame, subject_id):
    
    observed_rows = len(frame)
    if observed_rows < 1:
        raise RuntimeError(f'{subject_id}: baseline workbook has no data row.')
    if observed_rows == 1:
        return frame.iloc[0], {
            'baseline_rows_observed': 1,
            'baseline_unique_rows': 1,
            'row_selection_method': 'single_row',
            'selected_source_row_index': str(frame.index[0]),
            'duplicate_rows_collapsed': 0,
        }

    
    
    
    unique_frame = frame.drop_duplicates(keep='first')
    if len(unique_frame) == 1:
        return unique_frame.iloc[0], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': 1,
            'row_selection_method': 'exact_duplicate_rows_collapsed',
            'selected_source_row_index': str(unique_frame.index[0]),
            'duplicate_rows_collapsed': observed_rows - 1,
        }

    normalized_to_original = {
        normalize_label(column): str(column) for column in unique_frame.columns
    }

    
    
    
    
    identifier_columns = [
        normalized_to_original[name]
        for name in ['identifier', 'subjectid', 'participantid', 'patientid']
        if name in normalized_to_original
    ]
    target_digits = ''.join(re.findall(r'\d+', subject_id)).lstrip('0') or '0'
    matched_indices = []
    for row_index, candidate in unique_frame.iterrows():
        for column in identifier_columns:
            value_digits = ''.join(re.findall(r'\d+', str(candidate[column])))
            value_digits = value_digits.lstrip('0') or '0'
            if value_digits == target_digits:
                matched_indices.append(row_index)
                break
    if len(matched_indices) == 1:
        selected_index = matched_indices[0]
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_subject_identifier_match',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }

    
    
    
    
    profiles = []
    for row_index, candidate in unique_frame.iterrows():
        valid_count = 0
        parse_errors = 0
        for _, aliases, parser_name in CLINICAL_FEATURE_SPECS:
            source_column = next(
                (
                    normalized_to_original[normalize_label(alias)]
                    for alias in aliases
                    if normalize_label(alias) in normalized_to_original
                ),
                None,
            )
            if source_column is None:
                continue
            try:
                parsed = PARSER_BY_NAME[parser_name](candidate[source_column])
            except Exception:
                parse_errors += 1
                continue
            if parsed is not None:
                valid_count += 1
        profiles.append({
            'row_index': row_index,
            'valid_selected_fields': valid_count,
            'parse_errors': parse_errors,
        })

    ordered = sorted(
        profiles,
        key=lambda item: (item['parse_errors'], -item['valid_selected_fields']),
    )
    best = ordered[0]
    runner_up = ordered[1]
    if (
        best['parse_errors'] == 0
        and best['valid_selected_fields']
        >= runner_up['valid_selected_fields'] + 3
    ):
        selected_index = best['row_index']
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_parse_complete_patient_row',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }

    profile_text = '; '.join(
        f"row={item['row_index']},valid={item['valid_selected_fields']},"
        f"errors={item['parse_errors']}"
        for item in profiles
    )
    raise RuntimeError(
        f'{subject_id}: {observed_rows} baseline rows remain ambiguous after '
        f'exact-deduplication ({len(unique_frame)} unique). Profiles: '
        f'{profile_text}. No row was selected.'
    )

clinical_exclusion_df = pd.DataFrame([
    {'source_field': name, 'exclusion_reason': reason, 'model_input': False}
    for name, reason in EXCLUDED_CLINICAL_SPECS
])
atomic_write_csv(clinical_exclusion_df, CLINICAL_EXCLUSION_AUDIT_PATH)






allowed_table_suffixes = {'.xlsx', '.xls', '.csv', '.tsv'}
baseline_candidates = sorted(
    path for path in RAW_DATA_ROOT.rglob('*demographic_baseline*')
    if path.is_file() and path.suffix.lower() in allowed_table_suffixes
)
manifest_subjects = set(manifest_df['subject_id'].astype(str))
baseline_by_subject = {}
unmapped_baseline_files = []
for path in baseline_candidates:
    match = re.search(r'(sub-stroke\d+)', str(path), flags=re.IGNORECASE)
    if match is None:
        unmapped_baseline_files.append(str(path))
        continue
    subject_id = match.group(1).lower()
    if subject_id not in manifest_subjects:
        continue
    if subject_id in baseline_by_subject:
        raise RuntimeError(
            f'{subject_id}: multiple baseline clinical workbooks found.'
        )
    baseline_by_subject[subject_id] = path

clinical_feature_records = {
    name: {
        'model_feature': name,
        'aliases': aliases,
        'parser': parser_name,
        'subjects_total': 0,
        'source_columns_resolved': 0,
        'subjects_present': 0,
        'subjects_missing': 0,
        'parse_errors': 0,
        'parse_error_subjects': [],
        'parse_error_value_examples': [],
        'train_present': 0,
        'val_present': 0,
        'test_present': 0,
        'resolved_source_columns': set(),
    }
    for name, aliases, parser_name in CLINICAL_FEATURE_SPECS
}

subject_clinical_rows = []
clinical_schema_sets = []
nihss_subscore_columns = set()
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id']).lower()
    split = str(manifest_row['dev_split'])
    baseline_path = baseline_by_subject.get(subject_id)
    if baseline_path is None:
        subject_clinical_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'baseline_clinical_path': '',
            'baseline_clinical_sha256': '',
            'baseline_rows_observed': 0,
            'baseline_unique_rows': 0,
            'row_selection_method': '',
            'selected_source_row_index': '',
            'duplicate_rows_collapsed': 0,
            'baseline_columns': 0,
            'selected_features_present': 0,
            'selected_features_missing': len(CLINICAL_FEATURE_SPECS),
            'clinical_parse_errors': 0,
            'clinical_parse_error_features': '',
            'aggregate_admission_nihss_available': False,
            'nihss_subscore_count': 0,
        })
        continue

    frame = read_baseline_table(baseline_path)
    frame = frame.dropna(axis=0, how='all').dropna(axis=1, how='all')
    row, row_resolution = resolve_baseline_row(frame, subject_id)
    normalized_to_original = {
        normalize_label(column): str(column) for column in frame.columns
    }
    clinical_schema_sets.append(tuple(sorted(normalized_to_original)))
    subject_subscores = [
        str(column)
        for column in frame.columns
        if 'nihss' in normalize_label(column)
        and normalize_label(column) != normalize_label('NIHSS at admission')
    ]
    nihss_subscore_columns.update(subject_subscores)

    present_count = 0
    missing_count = 0
    parse_error_count = 0
    parse_error_features = []
    aggregate_nihss_available = False
    for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
        record = clinical_feature_records[model_name]
        record['subjects_total'] += 1
        source_column = next(
            (
                normalized_to_original[normalize_label(alias)]
                for alias in aliases
                if normalize_label(alias) in normalized_to_original
            ),
            None,
        )
        if source_column is None:
            record['subjects_missing'] += 1
            missing_count += 1
            continue
        record['source_columns_resolved'] += 1
        record['resolved_source_columns'].add(source_column)
        value = row[source_column]
        try:
            parsed = PARSER_BY_NAME[parser_name](value)
        except Exception:
            parsed = None
            record['parse_errors'] += 1
            record['parse_error_subjects'].append(subject_id)
            if len(record['parse_error_value_examples']) < 5:
                record['parse_error_value_examples'].append(repr(value))
            parse_error_count += 1
            parse_error_features.append(model_name)
        if parsed is None:
            record['subjects_missing'] += 1
            missing_count += 1
        else:
            record['subjects_present'] += 1
            record[f'{split}_present'] += 1
            present_count += 1
            if model_name == 'nihss_at_admission':
                aggregate_nihss_available = True

    subject_clinical_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'baseline_clinical_path': str(baseline_path),
        'baseline_clinical_sha256': file_sha256(baseline_path),
        **row_resolution,
        'baseline_columns': len(frame.columns),
        'selected_features_present': present_count,
        'selected_features_missing': missing_count,
        'clinical_parse_errors': parse_error_count,
        'clinical_parse_error_features': ' | '.join(parse_error_features),
        'aggregate_admission_nihss_available': aggregate_nihss_available,
        'nihss_subscore_count': len(subject_subscores),
    })

clinical_rows = []
for model_name, aliases, parser_name in CLINICAL_FEATURE_SPECS:
    record = clinical_feature_records[model_name]
    clinical_rows.append({
        'model_feature': model_name,
        'source_aliases': ' | '.join(aliases),
        'resolved_source_columns': ' | '.join(
            sorted(record['resolved_source_columns'])
        ),
        'parser': parser_name,
        'subjects_total': record['subjects_total'],
        'source_columns_resolved': record['source_columns_resolved'],
        'subjects_present': record['subjects_present'],
        'subjects_missing': record['subjects_missing'],
        'missing_fraction': (
            record['subjects_missing'] / max(record['subjects_total'], 1)
        ),
        'parse_errors': record['parse_errors'],
        'parse_error_subjects': ' | '.join(record['parse_error_subjects']),
        'parse_error_value_examples': ' | '.join(
            record['parse_error_value_examples']
        ),
        'train_present': record['train_present'],
        'val_present': record['val_present'],
        'test_present': record['test_present'],
        'paired_missingness_indicator': True,
        'scaler_policy': (
            'fit_on_development_fold_train_only_then_freeze'
            if parser_name in {'numeric', 'duration_hours'}
            else 'not_scaled_binary'
        ),
    })

clinical_feature_df = pd.DataFrame(clinical_rows)
clinical_subject_df = pd.DataFrame(subject_clinical_rows)
raw_baseline_schema_variant_count = len(set(clinical_schema_sets))
clinical_parse_error_count = int(clinical_feature_df['parse_errors'].sum())
clinical_resolved_value_count = int(
    clinical_feature_df['source_columns_resolved'].sum()
)
clinical_parse_error_fraction = (
    clinical_parse_error_count / max(clinical_resolved_value_count, 1)
)
atomic_write_csv(clinical_feature_df, CLINICAL_FEATURE_AUDIT_PATH)






node_cta_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('cta_')
]
node_tmax_indices = [
    index for index, name in enumerate(node_feature_names)
    if name.startswith('tmax_')
]
vessel_cta_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name in {'cta_mean', 'cta_p90'}
]
vessel_tmax_indices = [
    index for index, name in enumerate(vessel_feature_names)
    if name == 'tmax_mean'
]

vascular_rows = []
subject_vascular_rows = []
for manifest_row in manifest_df.to_dict('records'):
    subject_id = str(manifest_row['subject_id'])
    split = str(manifest_row['dev_split'])
    npz_path = Path(str(manifest_row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(manifest_row['trial8_graph_sidecar_path']))
    if not npz_path.exists():
        raise FileNotFoundError(f'{subject_id}: NPZ is missing: {npz_path}')
    if not sidecar_path.exists():
        raise FileNotFoundError(
            f'{subject_id}: graph sidecar is missing: {sidecar_path}'
        )
    if file_sha256(sidecar_path) != str(
        manifest_row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')

    with np.load(npz_path, allow_pickle=False) as data:
        npz_keys = set(data.files)
        missing_safe_keys = sorted(
            {'input_channels', *VASCULAR_PRIOR_ARRAYS} - npz_keys
        )
        input_channels = (
            decode_strings(data['input_channels'])
            if 'input_channels' in npz_keys
            else []
        )
        lvo_present = (
            bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            if 'lvo_present' in npz_keys
            else False
        )
        vessel_presence_count = (
            int((np.asarray(data['vessel_presence']) > 0).sum())
            if 'vessel_presence' in npz_keys
            else 0
        )

    with np.load(sidecar_path, allow_pickle=False) as data:
        sidecar_keys = set(data.files)
        forbidden_in_sidecar = sorted(FORBIDDEN_TARGET_KEYS & sidecar_keys)
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0

    atlas_enabled = as_bool(manifest_row['trial8_atlas_enabled'])
    valid_territory_nodes = int(node_valid.sum())
    valid_vessel_nodes = int(vessel_valid.sum())
    territory_cta_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_cta_indices]).sum()
    )
    territory_tmax_support = int(
        (node_valid[:, None] & node_feature_valid[:, node_tmax_indices]).sum()
    )
    vessel_cta_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_cta_indices]).sum()
    )
    vessel_tmax_support = int(
        (vessel_valid[:, None] & vessel_feature_valid[:, vessel_tmax_indices]).sum()
    )
    subject_vascular_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'input_channel_schema_matches': input_channels == EXPECTED_INPUT_CHANNELS,
        'cta_channel_available': 'cta' in input_channels,
        'tmax_channel_available': 'tmax' in input_channels,
        'ttp_channel_available': 'ttp' in input_channels,
        'perfusion_valid_channel_available': 'perfusion_valid' in input_channels,
        'cow_soft_union_available': 'cow_soft_union' in npz_keys,
        'lvo_proximity_available': 'lvo_proximity' in npz_keys,
        'lvo_present_field_available': 'lvo_present' in npz_keys,
        'lvo_present': lvo_present,
        'vessel_presence_field_available': 'vessel_presence' in npz_keys,
        'raw_vessel_presence_count': vessel_presence_count,
        'missing_safe_npz_keys': ' | '.join(missing_safe_keys),
        'sidecar_contains_forbidden_target_key': bool(forbidden_in_sidecar),
        'forbidden_sidecar_keys': ' | '.join(forbidden_in_sidecar),
        'atlas_enabled': atlas_enabled,
        'valid_territory_nodes': valid_territory_nodes,
        'valid_vessel_nodes': valid_vessel_nodes,
        'territory_cta_support': territory_cta_support,
        'territory_tmax_support': territory_tmax_support,
        'vessel_cta_support': vessel_cta_support,
        'vessel_tmax_support': vessel_tmax_support,
        'spatial_vascular_stream_available': (
            input_channels == EXPECTED_INPUT_CHANNELS
            and not missing_safe_keys
        ),
        'territory_graph_available': (
            atlas_enabled and valid_territory_nodes > 0
        ),
        'vessel_graph_available': valid_vessel_nodes > 0,
        'occlusion_source_valid': lvo_present,
    })

vascular_subject_df = pd.DataFrame(subject_vascular_rows)
for split, group in vascular_subject_df.groupby('dev_split', sort=True):
    vascular_rows.append({
        'dev_split': split,
        'subjects': len(group),
        'spatial_vascular_stream_available': int(
            group['spatial_vascular_stream_available'].sum()
        ),
        'territory_graph_available': int(
            group['territory_graph_available'].sum()
        ),
        'vessel_graph_available': int(group['vessel_graph_available'].sum()),
        'occlusion_source_valid': int(group['occlusion_source_valid'].sum()),
        'minimum_valid_territory_nodes': int(
            group['valid_territory_nodes'].min()
        ),
        'minimum_valid_vessel_nodes': int(group['valid_vessel_nodes'].min()),
        'mean_valid_vessel_nodes': float(group['valid_vessel_nodes'].mean()),
    })
vascular_audit_df = pd.DataFrame(vascular_rows)
atomic_write_csv(vascular_audit_df, VASCULAR_AUDIT_PATH)

subject_audit_df = clinical_subject_df.merge(
    vascular_subject_df,
    on=['subject_id', 'dev_split'],
    how='outer',
    validate='one_to_one',
)
atomic_write_csv(subject_audit_df, SUBJECT_AUDIT_PATH)






clinical_feature_count = len(CLINICAL_FEATURE_SPECS)
clinical_token_input_dim = clinical_feature_count * 2

architecture_contract = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'architecture_version': REVISED_MODEL_VERSION,
    'status': 'input_contract_only_model_not_constructed',
    'preserved_reference_path': {
        'image_backbone': '15-channel compact residual 3D U-Net',
        'graph_encoder': 'heterogeneous relation-aware GATv2',
        'graph_modulation': 'bounded zero-init multiscale FiLM',
        'reference_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    },
    'asymmetric_vascular_stream': {
        'spatial_inputs': VASCULAR_IMAGE_CHANNELS + [
            'cow_soft_union', 'lvo_proximity'
        ],
        'planned_channels': [8, 16, 32, 48],
        'planned_fusion_scales': ['quarter', 'eighth', 'sixteenth', 'bottleneck'],
        'fusion': 'confidence_gated_zero_initialized_residual_adapters',
        'confidence_inputs': [
            'perfusion_valid',
            'cow_prior_valid',
            'registration_confidence',
            'lvo_present',
        ],
        'purpose': (
            'emphasize CTA/perfusion/arterial structure without duplicating '
            'the full image backbone'
        ),
    },
    'occlusion_source': {
        'representation': 'validity_masked_source_token_or_graph_node',
        'inputs': [
            'lvo_present',
            'lvo_proximity',
            'vessel_cta_mean',
            'vessel_cta_p90',
            'vessel_tmax_mean',
            'vessel_minimum_lvo_distance_mm',
            'vessel_lvo_overlap_fraction',
        ],
        'ttp_used': False,
        'ttp_reason': 'TTP is not present in the frozen 15-channel input schema',
        'tmax_used': True,
        'invalid_state': 'zero token plus explicit validity bit',
        'external_deployment_dependency': (
            'requires the provided baseline LVO annotation or a separately '
            'validated CTA-derived occlusion detector'
        ),
    },
    'clinical_token': {
        'source': 'ses-01 demographic_baseline only',
        'selected_scalar_features': [
            name for name, _, _ in CLINICAL_FEATURE_SPECS
        ],
        'selected_scalar_count': clinical_feature_count,
        'missingness_indicator_count': clinical_feature_count,
        'input_dimension': clinical_token_input_dim,
        'planned_embedding_dimension': 32,
        'nihss_policy': 'aggregate admission NIHSS only; no subscores available',
        'missing_data_policy': (
            'fold-train median for continuous features, neutral zero for '
            'binary features, and one explicit missingness bit per scalar'
        ),
        'invalid_value_policy': (
            'unparseable values are quarantined as missing, receive the same '
            'explicit missingness bit, and are never guessed or coerced'
        ),
        'clinical_parse_error_count': clinical_parse_error_count,
        'clinical_parse_error_fraction_of_resolved_values': (
            clinical_parse_error_fraction
        ),
        'scaler_policy': 'fit inside each development fold train partition only',
        'raw_baseline_schema_variant_count': raw_baseline_schema_variant_count,
        'raw_schema_variation_policy': (
            'informational only; every selected source field must still resolve '
            'for every subject'
        ),
        'excluded_fields': clinical_exclusion_df.to_dict('records'),
    },
    'anti_leakage': {
        'outcome_workbooks_read': False,
        'target_arrays_read': False,
        'center_used': False,
        'post_imaging_treatment_times_used': False,
        'post_treatment_outcomes_used': False,
        'clinical_scaler_fitted_in_this_audit': False,
    },
    'regularization_contract': {
        'zero_initialized_new_fusion_paths': True,
        'independent_vascular_branch_dropout': True,
        'clinical_token_dropout': True,
        'atlas_and_occlusion_source_dropout': True,
        'missing_inputs_use_explicit_validity_masks': True,
        'direct_graph_to_logit_residual': False,
        'expert_routing': False,
    },
    'parameter_budget': {
        'baseline_parameters': EXPECTED_BASELINE_PARAMETERS,
        'planned_total_minimum': 15_300_000,
        'planned_total_maximum': 15_650_000,
        'exact_count_required_in_next_model_preflight': True,
    },
    'architecture_implementation_authorized': True,
    'model_training_authorized': False,
}

clinical_checks = {
    'exactly_one_baseline_workbook_per_subject': (
        len(baseline_by_subject) == EXPECTED_SUBJECTS
        and clinical_subject_df['baseline_clinical_path'].astype(str).str.len().gt(0).all()
    ),
    'all_baseline_workbooks_resolve_to_one_unambiguous_patient_row': bool(
        clinical_subject_df['row_selection_method'].fillna('').str.len().gt(0).all()
    ),
    
    
    
    'all_selected_clinical_features_have_resolved_source_support': bool(
        clinical_feature_df['source_columns_resolved'].gt(0).all()
    ),
    'all_selected_features_have_train_support': bool(
        clinical_feature_df['train_present'].gt(0).all()
    ),
    'all_selected_clinical_values_are_accounted_for': bool(
        (
            clinical_feature_df['subjects_present']
            + clinical_feature_df['subjects_missing']
        ).eq(clinical_feature_df['subjects_total']).all()
    ),
    'all_unparseable_values_are_quarantined_as_missing': bool(
        clinical_feature_df['parse_errors']
        .le(clinical_feature_df['subjects_missing']).all()
        and clinical_parse_error_count
        == int(clinical_subject_df['clinical_parse_errors'].sum())
    ),
    'clinical_parse_error_burden_is_below_one_percent_per_feature': bool(
        (
            clinical_feature_df['parse_errors']
            / clinical_feature_df['source_columns_resolved'].clip(lower=1)
        ).le(0.01).all()
    ),
    'aggregate_admission_nihss_available': bool(
        clinical_feature_df.loc[
            clinical_feature_df['model_feature'] == 'nihss_at_admission',
            'train_present',
        ].gt(0).all()
    ),
    'nihss_subscores_are_not_available_or_claimed': (
        len(nihss_subscore_columns) == 0
        and clinical_subject_df['nihss_subscore_count'].eq(0).all()
    ),
    'center_is_excluded': 'center' not in {
        name for name, _, _ in CLINICAL_FEATURE_SPECS
    },
    'post_imaging_treatment_fields_are_excluded': all(
        normalize_label(name) not in selected_normalized_names
        for name in [
            'Door to groin', 'Door to first series',
            'Time of intervention', 'Door to recanalization',
        ]
    ),
    'outcome_workbooks_not_read': True,
    'clinical_scaler_not_fitted': True,
}

vascular_checks = {
    'cta_tmax_and_perfusion_valid_channels_available_for_all': bool(
        vascular_subject_df[[
            'cta_channel_available',
            'tmax_channel_available',
            'perfusion_valid_channel_available',
        ]].all().all()
    ),
    'input_channel_schema_matches_for_all': bool(
        vascular_subject_df['input_channel_schema_matches'].all()
    ),
    'ttp_absent_and_tmax_selected': (
        not bool(vascular_subject_df['ttp_channel_available'].any())
        and bool(vascular_subject_df['tmax_channel_available'].all())
    ),
    'cow_and_lvo_prior_fields_available_for_all': bool(
        vascular_subject_df[[
            'cow_soft_union_available',
            'lvo_proximity_available',
            'lvo_present_field_available',
            'vessel_presence_field_available',
        ]].all().all()
    ),
    'spatial_vascular_stream_available_for_all': bool(
        vascular_subject_df['spatial_vascular_stream_available'].all()
    ),
    'graph_sidecars_contain_no_target_keys': not bool(
        vascular_subject_df['sidecar_contains_forbidden_target_key'].any()
    ),
    'all_atlas_enabled_subjects_have_territory_cta_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_cta_support'
        ].gt(0).all()
    ),
    'all_atlas_enabled_subjects_have_territory_tmax_support': bool(
        vascular_subject_df.loc[
            vascular_subject_df['atlas_enabled'], 'territory_tmax_support'
        ].gt(0).all()
    ),
    'vessel_cta_schema_present': len(vessel_cta_indices) == 2,
    'vessel_tmax_schema_present': len(vessel_tmax_indices) == 1,
    'missing_vascular_graph_inputs_have_validity_masks': bool(
        (
            (~vascular_subject_df['atlas_enabled'])
            | vascular_subject_df['territory_graph_available']
        ).all()
    ),
}

all_checks = {
    **upstream_checks,
    **clinical_checks,
    **vascular_checks,
    'clinical_token_dimension_is_44': clinical_token_input_dim == 44,
    'baseline_architecture_preserved_as_reference': True,
    'new_fusion_paths_planned_zero_initialized': True,
    'direct_graph_to_logit_residual_remains_disabled': True,
    'expert_routing_remains_disabled': True,
    'target_arrays_not_read': True,
    'locked_validation_and_test_inputs_used_for_availability_only': True,
    'study_model_not_constructed': True,
    'study_trainable_parameters_are_zero': True,
    'model_training_remains_locked': True,
}
audit_passed = bool(all(all_checks.values()))

architecture_contract['architecture_implementation_authorized'] = audit_passed
atomic_write_json(architecture_contract, ARCHITECTURE_CONTRACT_PATH)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'audit_version': AUDIT_VERSION,
    'feature_availability_audit_passed': audit_passed,
    'checks': all_checks,
    'manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'manifest_sha256': manifest_sha256,
    'graph_schema_path': str(GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': schema_sha256,
    'baseline_preflight_gate_path': str(BASELINE_PREFLIGHT_GATE_PATH),
    'clinical_feature_audit_path': str(CLINICAL_FEATURE_AUDIT_PATH),
    'clinical_exclusion_audit_path': str(CLINICAL_EXCLUSION_AUDIT_PATH),
    'vascular_feature_audit_path': str(VASCULAR_AUDIT_PATH),
    'subject_audit_path': str(SUBJECT_AUDIT_PATH),
    'architecture_contract_path': str(ARCHITECTURE_CONTRACT_PATH),
    'selected_clinical_scalar_features': clinical_feature_count,
    'clinical_token_input_dimension': clinical_token_input_dim,
    'raw_baseline_schema_variant_count': raw_baseline_schema_variant_count,
    'clinical_parse_error_count': clinical_parse_error_count,
    'clinical_parse_error_fraction_of_resolved_values': (
        clinical_parse_error_fraction
    ),
    'nihss_subscores_available': False,
    'aggregate_admission_nihss_used': True,
    'ttp_available': False,
    'tmax_used_for_occlusion_source': True,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'clinical_scaler_fitted': False,
    'study_model_constructed': False,
    'study_trainable_parameters': 0,
    'architecture_implementation_authorized': audit_passed,
    'model_training_authorized': False,
    'next_block': (
        'construct and smoke-test VascuFateNet-v2 with exact parameter, '
        'neutrality, ablation, gradient, and L4-memory checks'
    ),
}
atomic_write_json(gate, AUDIT_GATE_PATH)

print('\nPretreatment clinical feature availability:')
display(clinical_feature_df)
print('\nExplicitly excluded clinical fields:')
display(clinical_exclusion_df)
print('\nVascular availability by split:')
display(vascular_audit_df)
print('\nFeature-availability checks:')
display(pd.DataFrame({
    'check': list(all_checks.keys()),
    'passed': list(all_checks.values()),
}))

if not audit_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT FAILED. '
        'Do not construct or train the revised model. Failed checks: '
        + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCULAR-FUSION FEATURE-AVAILABILITY AUDIT PASSED')
print('=' * 80)
print(f'Subjects audited:                    {len(subject_audit_df)}')
print(f'Pretreatment clinical scalars:       {clinical_feature_count}')
print(f'Clinical missingness indicators:     {clinical_feature_count}')
print(f'Clinical-token input dimension:      {clinical_token_input_dim}')
print(f'Raw baseline schema variants:        {raw_baseline_schema_variant_count}')
print(f'Quarantined clinical parse errors:   {clinical_parse_error_count}')
print('Admission NIHSS:                     aggregate score available')
print('NIHSS subscores:                     unavailable; not used')
print('Occlusion timing signal:             Tmax (TTP unavailable)')
print('LVO source:                          validity-masked baseline annotation')
print('Outcome workbooks read:              False')
print('Target arrays read:                  False')
print('Clinical scaler fitted:              False')
print('Study trainable parameters:          0 (availability audit only)')
print(f'Baseline parameters preserved:       {EXPECTED_BASELINE_PARAMETERS:,}')
print(f'Architecture implementation allowed: {audit_passed}')
print('Model training authorized:           False')
print(f'Architecture contract:               {ARCHITECTURE_CONTRACT_PATH}')
print(f'Availability gate:                   {AUDIT_GATE_PATH}')
print(
    '\nNext: construct and smoke-test VascuFateNet-v2 with exact parameter, '
    'neutrality, ablation, gradient, and L4-memory checks.'
)


In [ ]:
import sys
import os
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AVAILABILITY_VERSION = 'v1_feature_availability_and_test_time_contract'
PREFLIGHT_VERSION = 'v1_zero_init_gated_fusion_construction'
CODE_REVISION = 'r1_v2_architecture_construction_preflight'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_BASELINE_PARAMETERS = 15_052_628

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

AVAILABILITY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / AVAILABILITY_VERSION
)
AVAILABILITY_GATE_PATH = AVAILABILITY_ROOT / 'feature_availability_gate.json'
INPUT_CONTRACT_PATH = AVAILABILITY_ROOT / 'architecture_input_contract.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = OUTPUT_ROOT / 'architecture_config.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_model_construction_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_model_construction_audit.csv'

SEED = 20260927
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_428_468
PARAMETER_BUDGET_MINIMUM = 15_300_000
PARAMETER_BUDGET_MAXIMUM = 15_650_000

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    AVAILABILITY_GATE_PATH,
    INPUT_CONTRACT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
availability_gate = json.loads(
    AVAILABILITY_GATE_PATH.read_text(encoding='utf-8')
)
input_contract = json.loads(INPUT_CONTRACT_PATH.read_text(encoding='utf-8'))
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')

if availability_gate.get('feature_availability_audit_passed') is not True:
    raise RuntimeError('Vascular-fusion feature-availability gate is not passed.')
if availability_gate.get('architecture_implementation_authorized') is not True:
    raise RuntimeError('Feature audit does not authorize architecture construction.')
if availability_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Feature audit must leave model training locked.')
if availability_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Feature audit did not remain target-free.')
if availability_gate.get('outcome_workbooks_read') is not False:
    raise RuntimeError('Feature audit read outcome workbooks.')
if availability_gate.get('manifest_sha256') != EXPECTED_TARGET_POLICY_MANIFEST_SHA256:
    raise RuntimeError('Feature audit target-policy manifest hash mismatch.')
if availability_gate.get('clinical_token_input_dimension') != CLINICAL_INPUT_DIM:
    raise RuntimeError('Clinical-token input dimension changed.')
if input_contract.get('architecture_version') != MODEL_VERSION:
    raise RuntimeError('Architecture input-contract version mismatch.')
if input_contract.get('model_training_authorized') is not False:
    raise RuntimeError('Input contract must leave model training locked.')
if input_contract.get('anti_leakage', {}).get('target_arrays_read') is not False:
    raise RuntimeError('Input contract did not remain target-free.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Vascular-fusion availability gate: passed')
print('Target arrays read: False')
print('Outcome workbooks read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

atomic_write_npz(
    SCALER_NPZ_PATH,
    territory_feature_names=np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    territory_mean=territory_mean,
    territory_std=territory_std,
    territory_count=territory_count,
    vessel_feature_names=np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    vessel_source_indices=np.asarray(vessel_source_indices, dtype=np.int16),
    vessel_mean=vessel_mean,
    vessel_std=vessel_std,
    vessel_count=vessel_count,
    case_feature_names=np.asarray(MODEL_CASE_FEATURE_NAMES),
    case_source_indices=np.asarray(case_source_indices, dtype=np.int16),
    case_scale_mask=case_scale_mask.astype(np.uint8),
    case_mean=case_mean,
    case_std=case_std,
    case_count=case_count,
    clip_value=np.asarray(SCALER_CLIP, dtype=np.float32),
    fit_split=np.asarray('train'),
    fit_subject_count=np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    fit_subject_hash=np.asarray(scaler_fit_subject_hash),
    target_arrays_read=np.asarray(False, dtype=np.uint8),
)
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(scaler_record, SCALER_JSON_PATH)

print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    

    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(
                math.atanh(float(fraction_of_bound))
            )


class AsymmetricVascularStream3D(nn.Module):
    

    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        pooled = v5.mean(dim=(2, 3, 4))
        return v3, v4, v5, pooled


class SpatialConfidenceFusion3D(nn.Module):
    

    def __init__(
        self,
        vascular_channels,
        image_channels,
        confidence_dim=4,
        bound=0.1,
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, kernel_size=1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound
        )

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature,
                size=image_feature.shape[2:],
                mode='trilinear',
                align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.value_dim = int(value_dim)
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        safe_values = values * valid
        return self.encoder(torch.cat([safe_values, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    

    def __init__(
        self,
        vascular_dim,
        occlusion_dim,
        clinical_dim,
        output_dim=64,
        confidence_dim=4,
        bound=0.1,
        dropout=0.10,
    ):
        super().__init__()
        source_dim = vascular_dim + occlusion_dim + clinical_dim
        self.source = nn.Sequential(
            nn.Linear(source_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self,
        graph_context,
        vascular_context,
        occlusion_context,
        clinical_context,
        confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context,
            occlusion_context,
            clinical_context,
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
        vascular_channels=(8, 16, 32, 48),
        clinical_input_dim=44,
        clinical_embed_dim=32,
        occlusion_value_dim=7,
        occlusion_embed_dim=32,
        confidence_dim=4,
        fusion_bound=0.1,
        vascular_dropout=0.15,
        clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)

        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        vc1, vc2, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vascular_dim=vc4,
            occlusion_dim=occlusion_embed_dim,
            clinical_dim=clinical_embed_dim,
            output_dim=graph_hidden,
            confidence_dim=confidence_dim,
            bound=fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        modules = [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]
        for module in modules:
            module.activate_for_gradient_test(0.5)

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        vascular_image,
        confidence_features,
        occlusion_values,
        occlusion_valid,
        clinical_input,
        vascular_available,
        occlusion_available,
        clinical_available,
        force_atlas_dropout=None,
        force_vascular_dropout=None,
        force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available,
            self.vascular_dropout,
            force_vascular_dropout,
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available,
            self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available,
            self.clinical_dropout,
            force_clinical_dropout,
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        vascular_gate = effective_vascular.to(image.dtype)
        safe_vascular_image = (
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        v3, v4, v5, vascular_context = self.vascular_stream(
            safe_vascular_image
        )
        vascular_context = (
            vascular_context * vascular_gate[:, None]
        )
        safe_occlusion_valid = (
            occlusion_valid
            * effective_occlusion[:, None].to(occlusion_valid.dtype)
        )
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input)
        clinical_context = (
            clinical_context * effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context,
            vascular_context,
            occlusion_context,
            clinical_context,
            effective_confidence,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')


preflight_row = None
for _, candidate_row in preflight_candidates.iterrows():
    candidate_npz_path = Path(str(candidate_row['vascufate_v2_npz_path']))
    with np.load(candidate_npz_path, allow_pickle=False) as candidate_data:
        candidate_lvo_present = (
            'lvo_present' in candidate_data.files
            and bool(np.asarray(candidate_data['lvo_present']).reshape(-1)[0])
        )
    if candidate_lvo_present:
        preflight_row = candidate_row
        break
if preflight_row is None:
    raise RuntimeError(
        'No atlas-enabled train-only case has a valid LVO source for the '
        'complete v2 branch smoke test.'
    )
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    required_npz_inputs = {
        'image',
        'subject_id',
        'input_channels',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
        'cow_prior_valid',
    }
    if not required_npz_inputs.issubset(data.files):
        raise RuntimeError('Preflight source NPZ lacks required input arrays.')
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
    input_channels = decode_strings(data['input_channels'])
    cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
    lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
    lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    cow_prior_valid = bool(
        np.asarray(data['cow_prior_valid']).reshape(-1)[0]
    )
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')
if input_channels != EXPECTED_INPUT_CHANNELS:
    raise RuntimeError('Preflight input-channel schema changed.')
if cow_soft_union.shape != image.shape[1:]:
    raise RuntimeError('CoW prior shape differs from the image grid.')
if lvo_proximity.shape != image.shape[1:]:
    raise RuntimeError('LVO proximity shape differs from the image grid.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))
cow_patch = center_crop_or_pad(
    cow_soft_union, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
lvo_patch = center_crop_or_pad(
    lvo_proximity, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
vascular_patch = np.stack([
    image_patch[input_channels.index('cta')],
    image_patch[input_channels.index('tmax')],
    image_patch[input_channels.index('perfusion_valid')],
    cow_patch,
    lvo_patch,
], axis=0).astype(np.float32)




clinical_input = np.zeros(CLINICAL_INPUT_DIM, dtype=np.float32)
clinical_input[:22] = np.linspace(-0.5, 0.5, 22, dtype=np.float32)
clinical_input[22:] = (np.arange(22) % 3 == 0).astype(np.float32)

occlusion_feature_names = [
    'cta_mean',
    'cta_p90',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
occlusion_values = [float(lvo_present), float(lvo_patch.max())]
occlusion_valid = [1.0, float(lvo_present)]
for feature_name in occlusion_feature_names:
    feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
    valid_nodes = selected_vessel_valid[:, feature_index]
    if bool(valid_nodes.any()):
        occlusion_values.append(float(scaled_vessel[valid_nodes, feature_index].mean()))
        occlusion_valid.append(1.0)
    else:
        occlusion_values.append(0.0)
        occlusion_valid.append(0.0)
occlusion_values = np.asarray(occlusion_values, dtype=np.float32)
occlusion_valid = np.asarray(occlusion_valid, dtype=np.float32)
if occlusion_values.shape != (OCCLUSION_VALUE_DIM,):
    raise RuntimeError('Occlusion-source vector dimension changed.')

confidence_features = np.asarray([
    float(vascular_patch[2].mean()),
    float(cow_prior_valid),
    float(np.clip(selected_case[1], 0.0, 1.0)),
    float(lvo_present),
], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
    'vascular_image': torch.from_numpy(vascular_patch[None]).to(device),
    'confidence_features': torch.from_numpy(
        confidence_features[None]
    ).to(device),
    'occlusion_values': torch.from_numpy(occlusion_values[None]).to(device),
    'occlusion_valid': torch.from_numpy(occlusion_valid[None]).to(device),
    'clinical_input': torch.from_numpy(clinical_input[None]).to(device),
    'vascular_available': torch.ones(1, dtype=torch.bool, device=device),
    'occlusion_available': torch.as_tensor(
        [lvo_present], dtype=torch.bool, device=device
    ),
    'clinical_available': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
vascular_stream_parameters = parameter_count(model.vascular_stream)
spatial_fusion_parameters = parameter_count(
    model.vascular_fusion_quarter
) + parameter_count(model.vascular_fusion_eighth) + parameter_count(
    model.vascular_fusion_bottleneck
)
clinical_parameters = parameter_count(model.clinical_encoder)
occlusion_parameters = parameter_count(model.occlusion_encoder)
context_fusion_parameters = parameter_count(model.context_fusion)
new_fusion_parameters = (
    vascular_stream_parameters
    + spatial_fusion_parameters
    + clinical_parameters
    + occlusion_parameters
    + context_fusion_parameters
)
backbone_parameters = total_parameters - (
    graph_parameters
    + film_parameters
    + head_parameters
    + new_fusion_parameters
)
preserved_v1_parameters = (
    backbone_parameters + graph_parameters + film_parameters + head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'asymmetric vascular stream', 'parameters': vascular_stream_parameters},
    {'component': 'confidence-gated spatial fusion', 'parameters': spatial_fusion_parameters},
    {'component': 'clinical token encoder', 'parameters': clinical_parameters},
    {'component': 'occlusion source encoder', 'parameters': occlusion_parameters},
    {'component': 'gated context fusion', 'parameters': context_fusion_parameters},
    {'component': 'preserved v1 path', 'parameters': preserved_v1_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad():
    normal_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    atlas_dropped_output = model(
        **batch,
        force_atlas_dropout=True,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    new_sources_dropped_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=True,
        force_clinical_dropout=True,
    )

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
    'fused_global_context': (1, GRAPH_HIDDEN),
    'new_source_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(new_sources_dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        atlas_dropped_output['atlas_dropped'].item()
        and not atlas_dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'],
        atlas_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'initial_segmentation_is_new_sources_neutral': torch.allclose(
        normal_output['logits'],
        new_sources_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
    'zero_init_new_fusion_gates_are_zero': all(
        torch.count_nonzero(module.network[-1].weight).item() == 0
        and torch.count_nonzero(module.network[-1].bias).item() == 0
        for module in [
            model.vascular_fusion_quarter.gate,
            model.vascular_fusion_eighth.gate,
            model.vascular_fusion_bottleneck.gate,
            model.context_fusion.gate,
        ]
    ),
    'normal_vascular_source_enabled': bool(
        normal_output['effective_vascular_enabled'].item()
    ),
    'normal_clinical_source_enabled': bool(
        normal_output['effective_clinical_enabled'].item()
    ),
    'forced_new_source_dropout_applied': bool(
        new_sources_dropped_output['vascular_dropped'].item()
        and new_sources_dropped_output['occlusion_dropped'].item()
        and new_sources_dropped_output['clinical_dropped'].item()
    ),
})




model.activate_new_fusion_gates_for_gradient_test()
with torch.no_grad():
    nn.init.normal_(model.global_film.project[-1].weight, mean=0.0, std=1e-3)
    nn.init.zeros_(model.global_film.project[-1].bias)

model.eval()
with torch.no_grad():
    activated_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    vascular_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    occlusion_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=True,
        force_clinical_dropout=False,
    )
    clinical_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=True,
    )

ablation_differences = {
    'vascular_max_abs_logit_change': float(
        (activated_output['logits'] - vascular_ablation_output['logits'])
        .abs().max().item()
    ),
    'occlusion_max_abs_logit_change': float(
        (activated_output['logits'] - occlusion_ablation_output['logits'])
        .abs().max().item()
    ),
    'clinical_max_abs_logit_change': float(
        (activated_output['logits'] - clinical_ablation_output['logits'])
        .abs().max().item()
    ),
}
ablation_checks = {
    'activated_outputs_finite': all(
        torch.isfinite(activated_output[name]).all().item()
        for name in expected_shapes
    ),
    'vascular_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['vascular_max_abs_logit_change'] > 1e-8
    ),
    'occlusion_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['occlusion_max_abs_logit_change'] > 1e-8
    ),
    'clinical_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['clinical_max_abs_logit_change'] > 1e-8
    ),
    'all_activated_gates_remain_bounded': all(
        float(activated_output[name].abs().max().item()) <= FUSION_BOUND + 1e-6
        for name in [
            'context_fusion_gate',
            'vascular_quarter_gate',
            'vascular_eighth_gate',
            'vascular_bottleneck_gate',
        ]
    ),
}

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=False,
):
    gradient_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('vascular_stream', model.vascular_stream),
    ('spatial_fusion', nn.ModuleList([
        model.vascular_fusion_quarter,
        model.vascular_fusion_eighth,
        model.vascular_fusion_bottleneck,
    ])),
    ('clinical_encoder', model.clinical_encoder),
    ('occlusion_encoder', model.occlusion_encoder),
    ('context_fusion', model.context_fusion),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'vascular_stream_has_gradients_after_gate_activation': (
        gradient_norms['vascular_stream'] > 0
    ),
    'spatial_fusion_has_gradients_after_gate_activation': (
        gradient_norms['spatial_fusion'] > 0
    ),
    'clinical_encoder_has_gradients_after_gate_activation': (
        gradient_norms['clinical_encoder'] > 0
    ),
    'occlusion_encoder_has_gradients_after_gate_activation': (
        gradient_norms['occlusion_encoder'] > 0
    ),
    'context_fusion_has_gradients_after_gate_activation': (
        gradient_norms['context_fusion'] > 0
    ),
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'image_channels': int(image.shape[0]),
    'patch_size': PATCH_SIZE,
    'backbone_channels': [32, 64, 128, 224, 320],
    'baseline_architecture_preserved': True,
    'baseline_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    'graph_variant': 'two_layer_relation_aware_gated_GATv2',
    'graph_hidden': GRAPH_HIDDEN,
    'graph_heads': GRAPH_HEADS,
    'graph_layers': GRAPH_LAYERS,
    'relation_bases': RELATION_BASES,
    'graph_dropout': GRAPH_DROPOUT,
    'edge_weight_use': 'log_attention_prior',
    'territory_nodes': NUM_TERRITORY_NODES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'conditioning': (
        'preserved_bounded_zero_initialized_multiscale_FiLM_plus_'
        'confidence_gated_zero_initialized_vascular_context_fusion'
    ),
    'film_bound': FILM_BOUND,
    'fusion_bound': FUSION_BOUND,
    'asymmetric_vascular_stream': {
        'input_channels': [
            'cta', 'tmax', 'perfusion_valid',
            'cow_soft_union', 'lvo_proximity',
        ],
        'channels': VASCULAR_CHANNELS,
        'fusion_scales': ['quarter', 'eighth', 'sixteenth'],
        'initial_gate': 'exact_zero',
    },
    'clinical_token': {
        'input_dimension': CLINICAL_INPUT_DIM,
        'embedding_dimension': CLINICAL_EMBED_DIM,
        'values': 22,
        'paired_missingness_indicators': 22,
        'fold_specific_scaler_required_before_training': True,
    },
    'occlusion_source': {
        'value_dimension': OCCLUSION_VALUE_DIM,
        'embedding_dimension': OCCLUSION_EMBED_DIM,
        'validity_masked': True,
        'uses_tmax_not_ttp': True,
    },
    'confidence_features': [
        'perfusion_valid_fraction',
        'cow_prior_valid',
        'registration_confidence',
        'lvo_present',
    ],
    'staged_neutrality': {
        'new_fusion_gates_exactly_zero_at_initialization': True,
        'controlled_gate_activation_used_only_for_preflight': True,
        'saved_or_trained_weights_created': False,
    },
    'graph_to_logit_residual': False,
    'expert_router': False,
    'deep_supervision': True,
    'territory_auxiliary_head': True,
    'scaler_policy': 'fit_on_train_only_then_freeze',
    'expected_total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
        'asymmetric_vascular_stream': vascular_stream_parameters,
        'spatial_fusion': spatial_fusion_parameters,
        'clinical_encoder': clinical_parameters,
        'occlusion_encoder': occlusion_parameters,
        'context_fusion': context_fusion_parameters,
        'new_fusion_total': new_fusion_parameters,
        'preserved_v1_total': preserved_v1_parameters,
    },
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_sha256': scaler_sha256,
    'feature_availability_gate_path': str(AVAILABILITY_GATE_PATH),
    'architecture_input_contract_path': str(INPUT_CONTRACT_PATH),
    'ablation_differences': ablation_differences,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'model_training_authorized': False,
}
atomic_write_json(architecture_config, ARCHITECTURE_JSON_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **forward_checks,
    **ablation_checks,
    **gradient_checks,
    'feature_availability_audit_passed': (
        availability_gate.get('feature_availability_audit_passed') is True
    ),
    'input_contract_authorizes_architecture_implementation': (
        input_contract.get('architecture_implementation_authorized') is True
    ),
    'preserved_v1_parameter_count_matches': (
        preserved_v1_parameters == EXPECTED_BASELINE_PARAMETERS
    ),
    'parameter_count_in_frozen_efficiency_budget': (
        PARAMETER_BUDGET_MINIMUM
        <= total_parameters
        <= PARAMETER_BUDGET_MAXIMUM
    ),
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'new_fusion_parameter_count_is_bounded': (
        0 < new_fusion_parameters < 600_000
    ),
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'direct_graph_to_logit_residual_disabled': True,
    'expert_routing_disabled': True,
    'training_remained_locked': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'ablation', 'check': name, 'passed': passed}
    for name, passed in ablation_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
 ] + [
    {'category': 'final_gate', 'check': name, 'passed': passed}
    for name, passed in preflight_checks.items()
    if name not in {
        *manifest_checks.keys(),
        *scaler_checks.keys(),
        *forward_checks.keys(),
        *ablation_checks.keys(),
        *gradient_checks.keys(),
    }
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'ablation_differences': ablation_differences,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'controlled_gate_activation_was_diagnostic_only': True,
    'saved_or_trained_weights_created': False,
    'v2_loader_and_clinical_scaler_implementation_authorized': preflight_passed,
    'model_training_authorized': False,
}
atomic_write_json(gate_record, PREFLIGHT_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 MODEL CONSTRUCTION PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'New fusion parameters:              {new_fusion_parameters:,}')
print(f'  Asymmetric vascular stream:       {vascular_stream_parameters:,}')
print(f'  Spatial confidence fusion:        {spatial_fusion_parameters:,}')
print(f'  Clinical encoder:                 {clinical_parameters:,}')
print(f'  Occlusion encoder:                {occlusion_parameters:,}')
print(f'  Context fusion:                   {context_fusion_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('New fusion initially neutral:       True')
print('Controlled ablations responsive:    True')
print('Graph scaler fitted on:             train only (102 subjects)')
print('Clinical scaler fitted:             False (next preflight)')
print('Target arrays read:                 False')
print('Outcome workbooks read:              False')
print('Model training authorized:          False')
print(f'V2 loader implementation allowed:   {preflight_passed}')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print(
    '\nNext: build the fold-specific clinical scaler and complete multimodal '
    'v2 loader preflight before any pilot training.'
)


In [ ]:
import sys
import os
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AVAILABILITY_VERSION = 'v1_feature_availability_and_test_time_contract'
PREFLIGHT_VERSION = 'v1_zero_init_gated_fusion_construction'
CODE_REVISION = 'r3_pytorch_module_namespace_safe'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_BASELINE_PARAMETERS = 15_052_628

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

AVAILABILITY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / AVAILABILITY_VERSION
)
AVAILABILITY_GATE_PATH = AVAILABILITY_ROOT / 'feature_availability_gate.json'
INPUT_CONTRACT_PATH = AVAILABILITY_ROOT / 'architecture_input_contract.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = OUTPUT_ROOT / 'architecture_config.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_model_construction_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_model_construction_audit.csv'

SEED = 20260927
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_428_468
PARAMETER_BUDGET_MINIMUM = 15_300_000
PARAMETER_BUDGET_MAXIMUM = 15_650_000

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    AVAILABILITY_GATE_PATH,
    INPUT_CONTRACT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
availability_gate = json.loads(
    AVAILABILITY_GATE_PATH.read_text(encoding='utf-8')
)
input_contract = json.loads(INPUT_CONTRACT_PATH.read_text(encoding='utf-8'))
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')

if availability_gate.get('feature_availability_audit_passed') is not True:
    raise RuntimeError('Vascular-fusion feature-availability gate is not passed.')
if availability_gate.get('architecture_implementation_authorized') is not True:
    raise RuntimeError('Feature audit does not authorize architecture construction.')
if availability_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Feature audit must leave model training locked.')
if availability_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Feature audit did not remain target-free.')
if availability_gate.get('outcome_workbooks_read') is not False:
    raise RuntimeError('Feature audit read outcome workbooks.')
if availability_gate.get('manifest_sha256') != EXPECTED_TARGET_POLICY_MANIFEST_SHA256:
    raise RuntimeError('Feature audit target-policy manifest hash mismatch.')
if availability_gate.get('clinical_token_input_dimension') != CLINICAL_INPUT_DIM:
    raise RuntimeError('Clinical-token input dimension changed.')
if input_contract.get('architecture_version') != MODEL_VERSION:
    raise RuntimeError('Architecture input-contract version mismatch.')
if input_contract.get('model_training_authorized') is not False:
    raise RuntimeError('Input contract must leave model training locked.')
if input_contract.get('anti_leakage', {}).get('target_arrays_read') is not False:
    raise RuntimeError('Input contract did not remain target-free.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print(f'Trial8 v2 model preflight revision: {CODE_REVISION}')
print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Vascular-fusion availability gate: passed')
print('Target arrays read: False')
print('Outcome workbooks read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

atomic_write_npz(
    SCALER_NPZ_PATH,
    territory_feature_names=np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    territory_mean=territory_mean,
    territory_std=territory_std,
    territory_count=territory_count,
    vessel_feature_names=np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    vessel_source_indices=np.asarray(vessel_source_indices, dtype=np.int16),
    vessel_mean=vessel_mean,
    vessel_std=vessel_std,
    vessel_count=vessel_count,
    case_feature_names=np.asarray(MODEL_CASE_FEATURE_NAMES),
    case_source_indices=np.asarray(case_source_indices, dtype=np.int16),
    case_scale_mask=case_scale_mask.astype(np.uint8),
    case_mean=case_mean,
    case_std=case_std,
    case_count=case_count,
    clip_value=np.asarray(SCALER_CLIP, dtype=np.float32),
    fit_split=np.asarray('train'),
    fit_subject_count=np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    fit_subject_hash=np.asarray(scaler_fit_subject_hash),
    target_arrays_read=np.asarray(False, dtype=np.uint8),
)
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(scaler_record, SCALER_JSON_PATH)

print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    

    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(
                math.atanh(float(fraction_of_bound))
            )


class AsymmetricVascularStream3D(nn.Module):
    

    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        pooled = v5.mean(dim=(2, 3, 4))
        return v3, v4, v5, pooled


class SpatialConfidenceFusion3D(nn.Module):
    

    def __init__(
        self,
        vascular_channels,
        image_channels,
        confidence_dim=4,
        bound=0.1,
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, kernel_size=1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound
        )

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature,
                size=image_feature.shape[2:],
                mode='trilinear',
                align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.value_dim = int(value_dim)
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        safe_values = values * valid
        return self.encoder(torch.cat([safe_values, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    

    def __init__(
        self,
        vascular_dim,
        occlusion_dim,
        clinical_dim,
        output_dim=64,
        confidence_dim=4,
        bound=0.1,
        dropout=0.10,
    ):
        super().__init__()
        source_dim = vascular_dim + occlusion_dim + clinical_dim
        self.source = nn.Sequential(
            nn.Linear(source_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self,
        graph_context,
        vascular_context,
        occlusion_context,
        clinical_context,
        confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context,
            occlusion_context,
            clinical_context,
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
        vascular_channels=(8, 16, 32, 48),
        clinical_input_dim=44,
        clinical_embed_dim=32,
        occlusion_value_dim=7,
        occlusion_embed_dim=32,
        confidence_dim=4,
        fusion_bound=0.1,
        vascular_dropout=0.15,
        clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)

        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        vc1, vc2, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vascular_dim=vc4,
            occlusion_dim=occlusion_embed_dim,
            clinical_dim=clinical_embed_dim,
            output_dim=graph_hidden,
            confidence_dim=confidence_dim,
            bound=fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        modules = [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]
        for module in modules:
            module.activate_for_gradient_test(0.5)

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        vascular_image,
        confidence_features,
        occlusion_values,
        occlusion_valid,
        clinical_input,
        vascular_available,
        occlusion_available,
        clinical_available,
        force_atlas_dropout=None,
        force_vascular_dropout=None,
        force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available,
            self.vascular_dropout,
            force_vascular_dropout,
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available,
            self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available,
            self.clinical_dropout,
            force_clinical_dropout,
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        vascular_gate = effective_vascular.to(image.dtype)
        safe_vascular_image = (
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        v3, v4, v5, vascular_context = self.vascular_stream(
            safe_vascular_image
        )
        vascular_context = (
            vascular_context * vascular_gate[:, None]
        )
        safe_occlusion_valid = (
            occlusion_valid
            * effective_occlusion[:, None].to(occlusion_valid.dtype)
        )
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input)
        clinical_context = (
            clinical_context * effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context,
            vascular_context,
            occlusion_context,
            clinical_context,
            effective_confidence,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')


preflight_row = None
for _, candidate_row in preflight_candidates.iterrows():
    candidate_npz_path = Path(str(candidate_row['vascufate_v2_npz_path']))
    with np.load(candidate_npz_path, allow_pickle=False) as candidate_data:
        candidate_lvo_present = (
            'lvo_present' in candidate_data.files
            and bool(np.asarray(candidate_data['lvo_present']).reshape(-1)[0])
        )
    if candidate_lvo_present:
        preflight_row = candidate_row
        break
if preflight_row is None:
    raise RuntimeError(
        'No atlas-enabled train-only case has a valid LVO source for the '
        'complete v2 branch smoke test.'
    )
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    required_npz_inputs = {
        'image',
        'subject_id',
        'input_channels',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
    }
    missing_npz_inputs = sorted(required_npz_inputs - set(data.files))
    if missing_npz_inputs:
        raise RuntimeError(
            'Preflight source NPZ lacks required input arrays: '
            + ', '.join(missing_npz_inputs)
        )
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
    input_channels = decode_strings(data['input_channels'])
    cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
    lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
    lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')
if input_channels != EXPECTED_INPUT_CHANNELS:
    raise RuntimeError('Preflight input-channel schema changed.')
if cow_soft_union.shape != image.shape[1:]:
    raise RuntimeError('CoW prior shape differs from the image grid.')
if lvo_proximity.shape != image.shape[1:]:
    raise RuntimeError('LVO proximity shape differs from the image grid.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

cow_prior_valid = bool(
    case_features_source[
        SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')
    ] > 0.5
)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))
cow_patch = center_crop_or_pad(
    cow_soft_union, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
lvo_patch = center_crop_or_pad(
    lvo_proximity, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
vascular_patch = np.stack([
    image_patch[input_channels.index('cta')],
    image_patch[input_channels.index('tmax')],
    image_patch[input_channels.index('perfusion_valid')],
    cow_patch,
    lvo_patch,
], axis=0).astype(np.float32)




clinical_input = np.zeros(CLINICAL_INPUT_DIM, dtype=np.float32)
clinical_input[:22] = np.linspace(-0.5, 0.5, 22, dtype=np.float32)
clinical_input[22:] = (np.arange(22) % 3 == 0).astype(np.float32)

occlusion_feature_names = [
    'cta_mean',
    'cta_p90',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
occlusion_values = [float(lvo_present), float(lvo_patch.max())]
occlusion_valid = [1.0, float(lvo_present)]
for feature_name in occlusion_feature_names:
    feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
    valid_nodes = selected_vessel_valid[:, feature_index]
    if bool(valid_nodes.any()):
        occlusion_values.append(float(scaled_vessel[valid_nodes, feature_index].mean()))
        occlusion_valid.append(1.0)
    else:
        occlusion_values.append(0.0)
        occlusion_valid.append(0.0)
occlusion_values = np.asarray(occlusion_values, dtype=np.float32)
occlusion_valid = np.asarray(occlusion_valid, dtype=np.float32)
if occlusion_values.shape != (OCCLUSION_VALUE_DIM,):
    raise RuntimeError('Occlusion-source vector dimension changed.')

confidence_features = np.asarray([
    float(vascular_patch[2].mean()),
    float(cow_prior_valid),
    float(np.clip(selected_case[1], 0.0, 1.0)),
    float(lvo_present),
], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
    'vascular_image': torch.from_numpy(vascular_patch[None]).to(device),
    'confidence_features': torch.from_numpy(
        confidence_features[None]
    ).to(device),
    'occlusion_values': torch.from_numpy(occlusion_values[None]).to(device),
    'occlusion_valid': torch.from_numpy(occlusion_valid[None]).to(device),
    'clinical_input': torch.from_numpy(clinical_input[None]).to(device),
    'vascular_available': torch.ones(1, dtype=torch.bool, device=device),
    'occlusion_available': torch.as_tensor(
        [lvo_present], dtype=torch.bool, device=device
    ),
    'clinical_available': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
vascular_stream_parameters = parameter_count(model.vascular_stream)
spatial_fusion_parameters = parameter_count(
    model.vascular_fusion_quarter
) + parameter_count(model.vascular_fusion_eighth) + parameter_count(
    model.vascular_fusion_bottleneck
)
clinical_parameters = parameter_count(model.clinical_encoder)
occlusion_parameters = parameter_count(model.occlusion_encoder)
context_fusion_parameters = parameter_count(model.context_fusion)
new_fusion_parameters = (
    vascular_stream_parameters
    + spatial_fusion_parameters
    + clinical_parameters
    + occlusion_parameters
    + context_fusion_parameters
)
backbone_parameters = total_parameters - (
    graph_parameters
    + film_parameters
    + head_parameters
    + new_fusion_parameters
)
preserved_v1_parameters = (
    backbone_parameters + graph_parameters + film_parameters + head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'asymmetric vascular stream', 'parameters': vascular_stream_parameters},
    {'component': 'confidence-gated spatial fusion', 'parameters': spatial_fusion_parameters},
    {'component': 'clinical token encoder', 'parameters': clinical_parameters},
    {'component': 'occlusion source encoder', 'parameters': occlusion_parameters},
    {'component': 'gated context fusion', 'parameters': context_fusion_parameters},
    {'component': 'preserved v1 path', 'parameters': preserved_v1_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad():
    normal_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    atlas_dropped_output = model(
        **batch,
        force_atlas_dropout=True,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    new_sources_dropped_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=True,
        force_clinical_dropout=True,
    )

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
    'fused_global_context': (1, GRAPH_HIDDEN),
    'new_source_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(new_sources_dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        atlas_dropped_output['atlas_dropped'].item()
        and not atlas_dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'],
        atlas_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'initial_segmentation_is_new_sources_neutral': torch.allclose(
        normal_output['logits'],
        new_sources_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
    'zero_init_new_fusion_gates_are_zero': all(
        torch.count_nonzero(module.network[-1].weight).item() == 0
        and torch.count_nonzero(module.network[-1].bias).item() == 0
        for module in [
            model.vascular_fusion_quarter.gate,
            model.vascular_fusion_eighth.gate,
            model.vascular_fusion_bottleneck.gate,
            model.context_fusion.gate,
        ]
    ),
    'normal_vascular_source_enabled': bool(
        normal_output['effective_vascular_enabled'].item()
    ),
    'normal_clinical_source_enabled': bool(
        normal_output['effective_clinical_enabled'].item()
    ),
    'forced_new_source_dropout_applied': bool(
        new_sources_dropped_output['vascular_dropped'].item()
        and new_sources_dropped_output['occlusion_dropped'].item()
        and new_sources_dropped_output['clinical_dropped'].item()
    ),
})




model.activate_new_fusion_gates_for_gradient_test()
with torch.no_grad():
    nn.init.normal_(model.global_film.project[-1].weight, mean=0.0, std=1e-3)
    nn.init.zeros_(model.global_film.project[-1].bias)

model.eval()
with torch.no_grad():
    activated_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    vascular_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    occlusion_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=True,
        force_clinical_dropout=False,
    )
    clinical_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=True,
    )

ablation_differences = {
    'vascular_max_abs_logit_change': float(
        (activated_output['logits'] - vascular_ablation_output['logits'])
        .abs().max().item()
    ),
    'occlusion_max_abs_logit_change': float(
        (activated_output['logits'] - occlusion_ablation_output['logits'])
        .abs().max().item()
    ),
    'clinical_max_abs_logit_change': float(
        (activated_output['logits'] - clinical_ablation_output['logits'])
        .abs().max().item()
    ),
}
ablation_checks = {
    'activated_outputs_finite': all(
        torch.isfinite(activated_output[name]).all().item()
        for name in expected_shapes
    ),
    'vascular_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['vascular_max_abs_logit_change'] > 1e-8
    ),
    'occlusion_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['occlusion_max_abs_logit_change'] > 1e-8
    ),
    'clinical_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['clinical_max_abs_logit_change'] > 1e-8
    ),
    'all_activated_gates_remain_bounded': all(
        float(activated_output[name].abs().max().item()) <= FUSION_BOUND + 1e-6
        for name in [
            'context_fusion_gate',
            'vascular_quarter_gate',
            'vascular_eighth_gate',
            'vascular_bottleneck_gate',
        ]
    ),
}

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=False,
):
    gradient_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('vascular_stream', model.vascular_stream),
    ('spatial_fusion', nn.ModuleList([
        model.vascular_fusion_quarter,
        model.vascular_fusion_eighth,
        model.vascular_fusion_bottleneck,
    ])),
    ('clinical_encoder', model.clinical_encoder),
    ('occlusion_encoder', model.occlusion_encoder),
    ('context_fusion', model.context_fusion),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'vascular_stream_has_gradients_after_gate_activation': (
        gradient_norms['vascular_stream'] > 0
    ),
    'spatial_fusion_has_gradients_after_gate_activation': (
        gradient_norms['spatial_fusion'] > 0
    ),
    'clinical_encoder_has_gradients_after_gate_activation': (
        gradient_norms['clinical_encoder'] > 0
    ),
    'occlusion_encoder_has_gradients_after_gate_activation': (
        gradient_norms['occlusion_encoder'] > 0
    ),
    'context_fusion_has_gradients_after_gate_activation': (
        gradient_norms['context_fusion'] > 0
    ),
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'image_channels': int(image.shape[0]),
    'patch_size': PATCH_SIZE,
    'backbone_channels': [32, 64, 128, 224, 320],
    'baseline_architecture_preserved': True,
    'baseline_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    'graph_variant': 'two_layer_relation_aware_gated_GATv2',
    'graph_hidden': GRAPH_HIDDEN,
    'graph_heads': GRAPH_HEADS,
    'graph_layers': GRAPH_LAYERS,
    'relation_bases': RELATION_BASES,
    'graph_dropout': GRAPH_DROPOUT,
    'edge_weight_use': 'log_attention_prior',
    'territory_nodes': NUM_TERRITORY_NODES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'conditioning': (
        'preserved_bounded_zero_initialized_multiscale_FiLM_plus_'
        'confidence_gated_zero_initialized_vascular_context_fusion'
    ),
    'film_bound': FILM_BOUND,
    'fusion_bound': FUSION_BOUND,
    'asymmetric_vascular_stream': {
        'input_channels': [
            'cta', 'tmax', 'perfusion_valid',
            'cow_soft_union', 'lvo_proximity',
        ],
        'channels': VASCULAR_CHANNELS,
        'fusion_scales': ['quarter', 'eighth', 'sixteenth'],
        'initial_gate': 'exact_zero',
    },
    'clinical_token': {
        'input_dimension': CLINICAL_INPUT_DIM,
        'embedding_dimension': CLINICAL_EMBED_DIM,
        'values': 22,
        'paired_missingness_indicators': 22,
        'fold_specific_scaler_required_before_training': True,
    },
    'occlusion_source': {
        'value_dimension': OCCLUSION_VALUE_DIM,
        'embedding_dimension': OCCLUSION_EMBED_DIM,
        'validity_masked': True,
        'uses_tmax_not_ttp': True,
    },
    'confidence_features': [
        'perfusion_valid_fraction',
        'cow_prior_valid',
        'registration_confidence',
        'lvo_present',
    ],
    'staged_neutrality': {
        'new_fusion_gates_exactly_zero_at_initialization': True,
        'controlled_gate_activation_used_only_for_preflight': True,
        'saved_or_trained_weights_created': False,
    },
    'graph_to_logit_residual': False,
    'expert_router': False,
    'deep_supervision': True,
    'territory_auxiliary_head': True,
    'scaler_policy': 'fit_on_train_only_then_freeze',
    'expected_total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
        'asymmetric_vascular_stream': vascular_stream_parameters,
        'spatial_fusion': spatial_fusion_parameters,
        'clinical_encoder': clinical_parameters,
        'occlusion_encoder': occlusion_parameters,
        'context_fusion': context_fusion_parameters,
        'new_fusion_total': new_fusion_parameters,
        'preserved_v1_total': preserved_v1_parameters,
    },
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_sha256': scaler_sha256,
    'feature_availability_gate_path': str(AVAILABILITY_GATE_PATH),
    'architecture_input_contract_path': str(INPUT_CONTRACT_PATH),
    'ablation_differences': ablation_differences,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'model_training_authorized': False,
}
atomic_write_json(architecture_config, ARCHITECTURE_JSON_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **forward_checks,
    **ablation_checks,
    **gradient_checks,
    'feature_availability_audit_passed': (
        availability_gate.get('feature_availability_audit_passed') is True
    ),
    'input_contract_authorizes_architecture_implementation': (
        input_contract.get('architecture_implementation_authorized') is True
    ),
    'preserved_v1_parameter_count_matches': (
        preserved_v1_parameters == EXPECTED_BASELINE_PARAMETERS
    ),
    'parameter_count_in_frozen_efficiency_budget': (
        PARAMETER_BUDGET_MINIMUM
        <= total_parameters
        <= PARAMETER_BUDGET_MAXIMUM
    ),
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'new_fusion_parameter_count_is_bounded': (
        0 < new_fusion_parameters < 600_000
    ),
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'direct_graph_to_logit_residual_disabled': True,
    'expert_routing_disabled': True,
    'training_remained_locked': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'ablation', 'check': name, 'passed': passed}
    for name, passed in ablation_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
 ] + [
    {'category': 'final_gate', 'check': name, 'passed': passed}
    for name, passed in preflight_checks.items()
    if name not in {
        *manifest_checks.keys(),
        *scaler_checks.keys(),
        *forward_checks.keys(),
        *ablation_checks.keys(),
        *gradient_checks.keys(),
    }
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'ablation_differences': ablation_differences,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'controlled_gate_activation_was_diagnostic_only': True,
    'saved_or_trained_weights_created': False,
    'v2_loader_and_clinical_scaler_implementation_authorized': preflight_passed,
    'model_training_authorized': False,
}
atomic_write_json(gate_record, PREFLIGHT_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 MODEL CONSTRUCTION PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'New fusion parameters:              {new_fusion_parameters:,}')
print(f'  Asymmetric vascular stream:       {vascular_stream_parameters:,}')
print(f'  Spatial confidence fusion:        {spatial_fusion_parameters:,}')
print(f'  Clinical encoder:                 {clinical_parameters:,}')
print(f'  Occlusion encoder:                {occlusion_parameters:,}')
print(f'  Context fusion:                   {context_fusion_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('New fusion initially neutral:       True')
print('Controlled ablations responsive:    True')
print('Graph scaler fitted on:             train only (102 subjects)')
print('Clinical scaler fitted:             False (next preflight)')
print('Target arrays read:                 False')
print('Outcome workbooks read:              False')
print('Model training authorized:          False')
print(f'V2 loader implementation allowed:   {preflight_passed}')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print(
    '\nNext: build the fold-specific clinical scaler and complete multimodal '
    'v2 loader preflight before any pilot training.'
)


In [ ]:
import sys
import os
import re
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, time, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            pip_name or import_name,
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')
ensure_package('openpyxl')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
RAW_DATA_ROOT = PROJECT_ROOT / 'ISLES24 dataset'
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
GRAPH_VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
BASELINE_PREFLIGHT_VERSION = 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
AVAILABILITY_VERSION = 'v1_feature_availability_and_test_time_contract'
MODEL_PREFLIGHT_VERSION = 'v1_zero_init_gated_fusion_construction'
PREFLIGHT_VERSION = 'v1_fold_clinical_scalers_multimodal_loader'
CODE_REVISION = 'r3_subject_index_safe_multimodal_loader'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_V2_PARAMETERS = 15_428_468
EXPECTED_CLINICAL_PARSE_ERRORS = 1
EXPECTED_CLINICAL_PARSE_ERROR_SUBJECT = 'sub-stroke0074'

NUMBER_OF_FOLDS = 5
PATCH_SIZE = (128, 128, 64)
CLINICAL_SCALER_CLIP = 5.0
GRAPH_SCALER_EPSILON = 1e-6
CLINICAL_SCALER_EPSILON = 1e-6
SEED = 20260927

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
    / 'verification'
    / GRAPH_VERIFICATION_VERSION
    / 'graph_schema_verified_pickle_free.npz'
)
BASELINE_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / BASELINE_PREFLIGHT_VERSION
)
BASELINE_PREFLIGHT_GATE_PATH = (
    BASELINE_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)
BASELINE_TRAINING_POLICY_PATH = (
    BASELINE_PREFLIGHT_ROOT / 'fivefold_training_policy.json'
)
AVAILABILITY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / AVAILABILITY_VERSION
)
AVAILABILITY_GATE_PATH = AVAILABILITY_ROOT / 'feature_availability_gate.json'
AVAILABILITY_SUBJECT_AUDIT_PATH = (
    AVAILABILITY_ROOT / 'subject_input_availability.csv'
)
MODEL_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / MODEL_PREFLIGHT_VERSION
)
MODEL_PREFLIGHT_GATE_PATH = (
    MODEL_PREFLIGHT_ROOT / 'v2_model_construction_gate.json'
)
MODEL_ARCHITECTURE_PATH = MODEL_PREFLIGHT_ROOT / 'architecture_config.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
CLINICAL_SIDECAR_ROOT = OUTPUT_ROOT / 'clinical_sidecars'
CLINICAL_SCALER_ROOT = OUTPUT_ROOT / 'fold_clinical_scalers'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CLINICAL_SIDECAR_ROOT.mkdir(parents=True, exist_ok=True)
CLINICAL_SCALER_ROOT.mkdir(parents=True, exist_ok=True)

CLINICAL_SCHEMA_PATH = OUTPUT_ROOT / 'clinical_schema.json'
CLINICAL_CASE_AUDIT_PATH = OUTPUT_ROOT / 'clinical_sidecar_case_audit.csv'
CLINICAL_SCALER_AUDIT_PATH = OUTPUT_ROOT / 'fold_clinical_scaler_audit.csv'
CLINICAL_TRANSFORM_AUDIT_PATH = (
    OUTPUT_ROOT / 'fold_clinical_transform_audit.csv'
)
LOADER_CASE_AUDIT_PATH = OUTPUT_ROOT / 'multimodal_loader_case_audit.csv'
LOADER_BATCH_AUDIT_PATH = OUTPUT_ROOT / 'multimodal_loader_batch_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_multimodal_loader_gate.json'
CANDIDATE_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}

CLINICAL_FEATURE_SPECS = [
    ('sex_female', ['Sex'], 'sex'),
    ('age_years', ['Age'], 'numeric'),
    ('atrial_fibrillation', ['Atrial fibrillation'], 'binary'),
    ('hypertension', ['Hypertension'], 'binary'),
    ('diabetes', ['Diabetes'], 'binary'),
    ('hyperlipidemia', ['Hyperlipidemia'], 'binary'),
    ('anticoagulation', ['Anticoagulation'], 'binary'),
    ('lipid_lowering_drugs', ['Lipid lowering drugs', 'Statins'], 'binary'),
    ('platelet_aggregation_inhibitors', ['PAIs'], 'binary'),
    ('glucose_mg_dl', ['Glucose'], 'numeric'),
    ('leucocytes_g_l', ['Leucocytes'], 'numeric'),
    ('crp_mg_dl', ['CRP'], 'numeric'),
    ('inr', ['INR'], 'numeric'),
    ('wake_up_stroke', ['Wake-up'], 'binary'),
    ('in_house_stroke', ['In-House', 'In-house'], 'binary'),
    ('external_referral', ['Referral'], 'binary'),
    ('onset_to_door_hours', ['Onset to door'], 'duration_hours'),
    ('alert_to_door_hours', ['Alert to door'], 'duration_hours'),
    ('nihss_at_admission', ['NIHSS at admission'], 'numeric'),
    ('mrs_at_admission', ['mRS at admission', 'MRS at admission'], 'numeric'),
    ('mrs_premorbid', ['mRS premorbid', 'MRS premorbid'], 'numeric'),
    ('door_to_imaging_hours', ['Door to imaging'], 'duration_hours'),
]
CLINICAL_FEATURE_NAMES = [item[0] for item in CLINICAL_FEATURE_SPECS]
CLINICAL_PARSER_NAMES = [item[2] for item in CLINICAL_FEATURE_SPECS]
CLINICAL_SCALE_MASK = np.asarray([
    parser_name in {'numeric', 'duration_hours'}
    for parser_name in CLINICAL_PARSER_NAMES
], dtype=bool)
CLINICAL_TOKEN_DIMENSION = len(CLINICAL_FEATURE_SPECS) * 2






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    output = []
    for value in np.asarray(array).reshape(-1):
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def subject_hash(subjects):
    payload = '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def normalize_label(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).strip().lower())


def is_missing(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip().lower() in {
        '', 'na', 'n/a', 'n.a.', 'nan', 'none', 'null', 'missing',
        'unknown', 'not available', 'not applicable', '-', '--',
    }:
        return True
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def parse_binary(value):
    if is_missing(value):
        return None
    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        numeric = float(value)
        if numeric in {0.0, 1.0}:
            return numeric
        raise ValueError('binary value is outside 0/1')
    text = str(value).strip().lower()
    mapping = {
        '0': 0.0, 'no': 0.0, 'false': 0.0, 'n': 0.0,
        '1': 1.0, 'yes': 1.0, 'true': 1.0, 'y': 1.0,
    }
    if text in mapping:
        return mapping[text]
    try:
        numeric = float(text)
    except ValueError as error:
        raise ValueError('unrecognized binary value') from error
    if numeric not in {0.0, 1.0}:
        raise ValueError('binary value is outside 0/1')
    return numeric


def parse_sex(value):
    if is_missing(value):
        return None
    text = str(value).strip().lower()
    mapping = {'f': 1.0, 'female': 1.0, 'm': 0.0, 'male': 0.0}
    if text not in mapping:
        raise ValueError('unrecognized sex value')
    return mapping[text]


def parse_numeric(value):
    if is_missing(value):
        return None
    numeric = float(value)
    if not math.isfinite(numeric):
        raise ValueError('non-finite numeric value')
    return numeric


def parse_duration_hours(value):
    if is_missing(value):
        return None
    if isinstance(value, pd.Timedelta):
        hours = value.total_seconds() / 3600.0
    elif isinstance(value, time):
        hours = (
            value.hour
            + value.minute / 60.0
            + value.second / 3600.0
            + value.microsecond / 3.6e9
        )
    elif isinstance(value, (int, float, np.integer, np.floating)):
        hours = float(value) * 24.0
    else:
        parsed = pd.to_timedelta(str(value).strip())
        hours = parsed.total_seconds() / 3600.0
    if not math.isfinite(hours) or hours < 0:
        raise ValueError('invalid duration')
    return hours


PARSER_BY_NAME = {
    'binary': parse_binary,
    'sex': parse_sex,
    'numeric': parse_numeric,
    'duration_hours': parse_duration_hours,
}


def read_baseline_table(path):
    suffix = Path(path).suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.tsv', '.txt'}:
        return pd.read_csv(path, sep='\t')
    raise RuntimeError(f'Unsupported baseline table format: {path}')


def resolve_baseline_row(frame, subject_id):
    
    observed_rows = len(frame)
    if observed_rows < 1:
        raise RuntimeError(f'{subject_id}: baseline workbook has no data row.')
    if observed_rows == 1:
        return frame.iloc[0], {
            'baseline_rows_observed': 1,
            'baseline_unique_rows': 1,
            'row_selection_method': 'single_row',
            'selected_source_row_index': str(frame.index[0]),
            'duplicate_rows_collapsed': 0,
        }

    unique_frame = frame.drop_duplicates(keep='first')
    if len(unique_frame) == 1:
        return unique_frame.iloc[0], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': 1,
            'row_selection_method': 'exact_duplicate_rows_collapsed',
            'selected_source_row_index': str(unique_frame.index[0]),
            'duplicate_rows_collapsed': observed_rows - 1,
        }

    normalized_to_original = {
        normalize_label(column): str(column) for column in unique_frame.columns
    }
    identifier_columns = [
        normalized_to_original[name]
        for name in ['identifier', 'subjectid', 'participantid', 'patientid']
        if name in normalized_to_original
    ]
    target_digits = ''.join(re.findall(r'\d+', subject_id)).lstrip('0') or '0'
    matched_indices = []
    for row_index, candidate in unique_frame.iterrows():
        for column in identifier_columns:
            value_digits = ''.join(re.findall(r'\d+', str(candidate[column])))
            value_digits = value_digits.lstrip('0') or '0'
            if value_digits == target_digits:
                matched_indices.append(row_index)
                break
    if len(matched_indices) == 1:
        selected_index = matched_indices[0]
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_subject_identifier_match',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }

    profiles = []
    for row_index, candidate in unique_frame.iterrows():
        valid_count = 0
        parse_errors = 0
        for _, aliases, parser_name in CLINICAL_FEATURE_SPECS:
            source_column = next((
                normalized_to_original[normalize_label(alias)]
                for alias in aliases
                if normalize_label(alias) in normalized_to_original
            ), None)
            if source_column is None:
                continue
            try:
                parsed = PARSER_BY_NAME[parser_name](candidate[source_column])
            except Exception:
                parse_errors += 1
                continue
            if parsed is not None:
                valid_count += 1
        profiles.append({
            'row_index': row_index,
            'valid_selected_fields': valid_count,
            'parse_errors': parse_errors,
        })
    ordered = sorted(
        profiles,
        key=lambda item: (
            item['parse_errors'], -item['valid_selected_fields']
        ),
    )
    best = ordered[0]
    runner_up = ordered[1]
    if (
        best['parse_errors'] == 0
        and best['valid_selected_fields']
        >= runner_up['valid_selected_fields'] + 3
    ):
        selected_index = best['row_index']
        return unique_frame.loc[selected_index], {
            'baseline_rows_observed': observed_rows,
            'baseline_unique_rows': len(unique_frame),
            'row_selection_method': 'unique_parse_complete_patient_row',
            'selected_source_row_index': str(selected_index),
            'duplicate_rows_collapsed': observed_rows - len(unique_frame),
        }
    profile_text = '; '.join(
        f"row={item['row_index']},valid={item['valid_selected_fields']},"
        f"errors={item['parse_errors']}"
        for item in profiles
    )
    raise RuntimeError(
        f'{subject_id}: {observed_rows} baseline rows remain ambiguous after '
        f'exact-deduplication ({len(unique_frame)} unique). Profiles: '
        f'{profile_text}. No row was selected.'
    )


def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std, clip_value):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -clip_value, clip_value)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)






required_paths = [
    TARGET_POLICY_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH,
    BASELINE_PREFLIGHT_GATE_PATH,
    BASELINE_TRAINING_POLICY_PATH,
    AVAILABILITY_GATE_PATH,
    AVAILABILITY_SUBJECT_AUDIT_PATH,
    MODEL_PREFLIGHT_GATE_PATH,
    MODEL_ARCHITECTURE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 v2 loader prerequisites are missing:\n' + '\n'.join(missing_paths)
    )

target_policy_manifest_sha256 = file_sha256(TARGET_POLICY_MANIFEST_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)
graph_schema_sha256 = file_sha256(GRAPH_SCHEMA_PATH)
if target_policy_manifest_sha256 != EXPECTED_TARGET_POLICY_MANIFEST_SHA256:
    raise RuntimeError('Target-policy manifest hash changed.')
if fold_manifest_sha256 != EXPECTED_FOLD_MANIFEST_SHA256:
    raise RuntimeError('Five-fold manifest hash changed.')
if graph_schema_sha256 != EXPECTED_GRAPH_SCHEMA_SHA256:
    raise RuntimeError('Verified graph schema hash changed.')

baseline_gate = json.loads(
    BASELINE_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
baseline_policy = json.loads(
    BASELINE_TRAINING_POLICY_PATH.read_text(encoding='utf-8')
)
availability_gate = json.loads(
    AVAILABILITY_GATE_PATH.read_text(encoding='utf-8')
)
model_gate = json.loads(
    MODEL_PREFLIGHT_GATE_PATH.read_text(encoding='utf-8')
)
model_architecture = json.loads(
    MODEL_ARCHITECTURE_PATH.read_text(encoding='utf-8')
)
model_architecture_sha256 = file_sha256(MODEL_ARCHITECTURE_PATH)

upstream_checks = {
    'baseline_fivefold_preflight_passed': baseline_gate.get('preflight_passed') is True,
    'baseline_loader_contract_verified': (
        baseline_gate.get('baseline_loader_loss_sampler_contract_verified') is True
    ),
    'baseline_target_policy_manifest_hash_matches': (
        baseline_gate.get('target_policy_manifest_sha256')
        == target_policy_manifest_sha256
    ),
    'baseline_fold_manifest_hash_matches': (
        baseline_gate.get('fold_manifest_sha256') == fold_manifest_sha256
    ),
    'baseline_training_remained_locked': (
        baseline_gate.get('fold0_pilot_training_authorized') is False
        and baseline_gate.get('full_fivefold_training_authorized') is False
    ),
    'baseline_policy_matches_gate_scalers': (
        baseline_policy.get('fold_scalers')
        == baseline_gate.get('fold_scaler_paths')
        and baseline_policy.get('fold_scaler_sha256')
        == baseline_gate.get('fold_scaler_sha256')
    ),
    'feature_availability_audit_passed': (
        availability_gate.get('feature_availability_audit_passed') is True
    ),
    'feature_audit_target_free': availability_gate.get('target_arrays_read') is False,
    'feature_audit_outcome_free': (
        availability_gate.get('outcome_workbooks_read') is False
    ),
    'v2_model_construction_passed': model_gate.get('preflight_passed') is True,
    'v2_loader_implementation_authorized': (
        model_gate.get('v2_loader_and_clinical_scaler_implementation_authorized')
        is True
    ),
    'v2_model_training_remained_locked': (
        model_gate.get('model_training_authorized') is False
    ),
    'v2_parameter_count_matches': (
        int(model_architecture.get('total_parameters', -1))
        == EXPECTED_V2_PARAMETERS
        and int(model_gate.get('total_parameters', -1))
        == EXPECTED_V2_PARAMETERS
    ),
    'v2_architecture_hash_matches_model_gate': (
        model_gate.get('architecture_config_sha256')
        == model_architecture_sha256
    ),
    'v2_architecture_requires_fold_clinical_scaler': (
        model_architecture.get('clinical_token', {}).get(
            'fold_specific_scaler_required_before_training'
        ) is True
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Trial8 v2 loader upstream checks failed: ' + ', '.join(failed))

manifest_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id', 'dev_split', 'vascufate_v2_npz_path',
    'trial8_npz_sha256', 'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
required_fold_columns = {
    'subject_id', 'frozen_dev_split', 'development_fold',
    'development_role', 'target_policy_manifest_sha256',
}
missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
if missing_manifest_columns or missing_fold_columns:
    raise RuntimeError(
        'Required manifest columns are missing. Policy missing: '
        f'{missing_manifest_columns}; fold missing: {missing_fold_columns}'
    )
manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
if manifest_df['subject_id'].astype(str).tolist() != fold_df['subject_id'].astype(str).tolist():
    raise RuntimeError('Policy and fold manifests have different subject order.')
manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
manifest_df['development_role'] = fold_df['development_role'].astype(str)

availability_df = pd.read_csv(AVAILABILITY_SUBJECT_AUDIT_PATH)
required_availability_columns = {
    'subject_id', 'dev_split', 'atlas_enabled', 'lvo_present',
    'valid_territory_nodes', 'valid_vessel_nodes',
}
missing_availability_columns = sorted(
    required_availability_columns - set(availability_df.columns)
)
if missing_availability_columns:
    raise RuntimeError(
        'Availability audit is missing columns: '
        + ', '.join(missing_availability_columns)
    )
availability_df['subject_id'] = availability_df['subject_id'].astype(str)
availability_by_subject = availability_df.set_index('subject_id')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS,
    'fold_splits_match_policy_manifest': fold_df[
        'frozen_dev_split'
    ].astype(str).equals(manifest_df['dev_split'].astype(str)),
    'fold_policy_hash_matches_for_all_rows': bool(
        fold_df['target_policy_manifest_sha256']
        .astype(str).eq(target_policy_manifest_sha256).all()
    ),
    'development_folds_are_zero_through_four': set(
        manifest_df.loc[manifest_df['dev_split'] == 'train', 'development_fold']
    ) == set(range(NUMBER_OF_FOLDS)),
    'validation_and_test_have_no_development_fold': bool(
        manifest_df.loc[
            manifest_df['dev_split'].isin(['val', 'test']), 'development_fold'
        ].eq(-1).all()
    ),
    'availability_audit_has_148_subjects': (
        availability_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Manifest checks failed: ' + ', '.join(failed))

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}
if any(value.dtype.kind == 'O' for value in graph_schema.values()):
    raise RuntimeError('Graph schema contains an object array.')

fold_graph_scaler_paths = {
    int(key): Path(value)
    for key, value in baseline_gate['fold_scaler_paths'].items()
}
fold_graph_scaler_hashes = {
    int(key): str(value)
    for key, value in baseline_gate['fold_scaler_sha256'].items()
}
if set(fold_graph_scaler_paths) != set(range(NUMBER_OF_FOLDS)):
    raise RuntimeError('Five graph scaler paths were not frozen.')
for fold_index, path in fold_graph_scaler_paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Fold-{fold_index} graph scaler is missing: {path}')
    if file_sha256(path) != fold_graph_scaler_hashes[fold_index]:
        raise RuntimeError(f'Fold-{fold_index} graph scaler hash mismatch.')

print(f'Trial8 v2 multimodal loader revision: {CODE_REVISION}')
print('All upstream data and architecture gates passed.')
print(f'Subjects:                           {len(manifest_df)}')
print(f'Split counts:                       {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'V2 architecture parameters:         {EXPECTED_V2_PARAMETERS:,}')
print('Target arrays read:                 False')
print('Outcome workbooks read:             False')
print('Study trainable parameters:         0 (loader preflight only)')


atomic_write_json({
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': False,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'training_performed': False,
    'study_trainable_parameters': 0,
    'fold0_pilot_training_authorized': False,
    'full_fivefold_training_authorized': False,
}, PREFLIGHT_GATE_PATH)






allowed_suffixes = {'.xlsx', '.xls', '.csv', '.tsv'}
baseline_candidates = sorted(
    path for path in RAW_DATA_ROOT.rglob('*demographic_baseline*')
    if path.is_file() and path.suffix.lower() in allowed_suffixes
)
manifest_subjects = set(manifest_df['subject_id'].astype(str))
baseline_by_subject = {}
for path in baseline_candidates:
    match = re.search(r'(sub-stroke\d+)', str(path), flags=re.IGNORECASE)
    if match is None:
        continue
    subject_id = match.group(1).lower()
    if subject_id not in manifest_subjects:
        continue
    if subject_id in baseline_by_subject:
        raise RuntimeError(f'{subject_id}: multiple baseline workbooks found.')
    baseline_by_subject[subject_id] = path
if set(baseline_by_subject) != manifest_subjects:
    missing = sorted(manifest_subjects - set(baseline_by_subject))
    raise RuntimeError('Missing baseline workbooks: ' + ', '.join(missing))

clinical_values_by_subject = {}
clinical_valid_by_subject = {}
clinical_sidecar_rows = []
clinical_parse_errors = []
for row in manifest_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    workbook_path = baseline_by_subject[subject_id]
    frame = read_baseline_table(workbook_path)
    frame = frame.dropna(axis=0, how='all').dropna(axis=1, how='all')
    clinical_row, row_resolution = resolve_baseline_row(
        frame, subject_id
    )
    normalized_to_original = {
        normalize_label(column): str(column) for column in frame.columns
    }
    values = np.zeros(len(CLINICAL_FEATURE_SPECS), dtype=np.float32)
    valid = np.zeros(len(CLINICAL_FEATURE_SPECS), dtype=bool)
    source_columns = []
    subject_error_features = []
    subject_error_values = []
    for feature_index, (feature_name, aliases, parser_name) in enumerate(
        CLINICAL_FEATURE_SPECS
    ):
        source_column = next((
            normalized_to_original[normalize_label(alias)]
            for alias in aliases
            if normalize_label(alias) in normalized_to_original
        ), None)
        source_columns.append(source_column or '')
        if source_column is None:
            continue
        raw_value = clinical_row[source_column]
        try:
            parsed = PARSER_BY_NAME[parser_name](raw_value)
        except Exception:
            clinical_parse_errors.append({
                'subject_id': subject_id,
                'feature_name': feature_name,
                'raw_value': repr(raw_value),
            })
            subject_error_features.append(feature_name)
            subject_error_values.append(repr(raw_value))
            parsed = None
        if parsed is not None:
            values[feature_index] = np.float32(parsed)
            valid[feature_index] = True

    sidecar_path = CLINICAL_SIDECAR_ROOT / f'{subject_id}_clinical_input.npz'
    atomic_write_npz(
        sidecar_path,
        subject_id=np.asarray(subject_id),
        feature_names=np.asarray(CLINICAL_FEATURE_NAMES, dtype='U'),
        parser_names=np.asarray(CLINICAL_PARSER_NAMES, dtype='U'),
        source_columns=np.asarray(source_columns, dtype='U'),
        values=values,
        valid=valid.astype(np.uint8),
        missing=(~valid).astype(np.uint8),
        baseline_workbook_sha256=np.asarray(file_sha256(workbook_path)),
        target_arrays_read=np.asarray(False, dtype=np.uint8),
        outcome_workbooks_read=np.asarray(False, dtype=np.uint8),
    )
    sidecar_sha256 = file_sha256(sidecar_path)
    with np.load(sidecar_path, allow_pickle=False) as saved:
        saved_values = np.asarray(saved['values'], dtype=np.float32)
        saved_valid = np.asarray(saved['valid']) > 0
        saved_names = decode_strings(saved['feature_names'])
        pickle_free = all(saved[name].dtype.kind != 'O' for name in saved.files)
    sidecar_integrity = (
        np.array_equal(saved_values, values)
        and np.array_equal(saved_valid, valid)
        and saved_names == CLINICAL_FEATURE_NAMES
        and pickle_free
    )
    clinical_values_by_subject[subject_id] = values
    clinical_valid_by_subject[subject_id] = valid
    clinical_sidecar_rows.append({
        'subject_id': subject_id,
        'dev_split': split,
        'development_fold': int(row['development_fold']),
        'baseline_workbook_path': str(workbook_path),
        'baseline_workbook_sha256': file_sha256(workbook_path),
        'baseline_rows_observed': row_resolution['baseline_rows_observed'],
        'baseline_unique_rows': row_resolution['baseline_unique_rows'],
        'row_resolution_method': row_resolution['row_selection_method'],
        'selected_source_row_index': row_resolution[
            'selected_source_row_index'
        ],
        'duplicate_rows_collapsed': row_resolution[
            'duplicate_rows_collapsed'
        ],
        'features_present': int(valid.sum()),
        'features_missing': int((~valid).sum()),
        'parse_error_count': len(subject_error_features),
        'parse_error_features': ' | '.join(subject_error_features),
        'parse_error_values': ' | '.join(subject_error_values),
        'clinical_sidecar_path': str(sidecar_path),
        'clinical_sidecar_sha256': sidecar_sha256,
        'sidecar_pickle_free': pickle_free,
        'sidecar_integrity_passed': sidecar_integrity,
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
    })

clinical_sidecar_df = pd.DataFrame(clinical_sidecar_rows)
atomic_write_csv(clinical_sidecar_df, CLINICAL_CASE_AUDIT_PATH)
clinical_sidecar_path_by_subject = {
    str(row['subject_id']): Path(str(row['clinical_sidecar_path']))
    for row in clinical_sidecar_rows
}
clinical_sidecar_hash_by_subject = {
    str(row['subject_id']): str(row['clinical_sidecar_sha256'])
    for row in clinical_sidecar_rows
}
clinical_schema = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'schema_version': PREFLIGHT_VERSION,
    'feature_names': CLINICAL_FEATURE_NAMES,
    'parser_names': CLINICAL_PARSER_NAMES,
    'scaled_features': [
        name for name, scaled in zip(CLINICAL_FEATURE_NAMES, CLINICAL_SCALE_MASK)
        if scaled
    ],
    'unscaled_binary_features': [
        name for name, scaled in zip(CLINICAL_FEATURE_NAMES, CLINICAL_SCALE_MASK)
        if not scaled
    ],
    'value_dimension': len(CLINICAL_FEATURE_NAMES),
    'missingness_dimension': len(CLINICAL_FEATURE_NAMES),
    'token_dimension': CLINICAL_TOKEN_DIMENSION,
    'missing_numeric_policy': 'fold_train_median_then_fold_train_zscore',
    'missing_binary_policy': 'neutral_zero_plus_missingness_indicator',
    'invalid_value_policy': 'quarantine_as_missing_never_coerce',
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
}
atomic_write_json(clinical_schema, CLINICAL_SCHEMA_PATH)
clinical_schema_sha256 = file_sha256(CLINICAL_SCHEMA_PATH)






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
validation_subjects = set(
    manifest_df.loc[manifest_df['dev_split'] == 'val', 'subject_id'].astype(str)
)
test_subjects = set(
    manifest_df.loc[manifest_df['dev_split'] == 'test', 'subject_id'].astype(str)
)
fold_clinical_scaler_paths = {}
fold_clinical_scaler_hashes = {}
clinical_scaler_rows = []
for heldout_fold in range(NUMBER_OF_FOLDS):
    fit_df = train_df[train_df['development_fold'] != heldout_fold].copy()
    heldout_df = train_df[train_df['development_fold'] == heldout_fold].copy()
    fit_subjects = fit_df['subject_id'].astype(str).tolist()
    heldout_subjects = heldout_df['subject_id'].astype(str).tolist()
    fit_subject_set = set(fit_subjects)
    heldout_subject_set = set(heldout_subjects)
    fit_values = np.stack([
        clinical_values_by_subject[subject_id] for subject_id in fit_subjects
    ])
    fit_valid = np.stack([
        clinical_valid_by_subject[subject_id] for subject_id in fit_subjects
    ])
    imputation = np.zeros(len(CLINICAL_FEATURE_NAMES), dtype=np.float32)
    mean = np.zeros(len(CLINICAL_FEATURE_NAMES), dtype=np.float32)
    std = np.ones(len(CLINICAL_FEATURE_NAMES), dtype=np.float32)
    count = fit_valid.sum(axis=0).astype(np.int32)
    constant = np.zeros(len(CLINICAL_FEATURE_NAMES), dtype=np.uint8)
    for feature_index, should_scale in enumerate(CLINICAL_SCALE_MASK):
        observed = fit_values[fit_valid[:, feature_index], feature_index]
        if len(observed) == 0:
            raise RuntimeError(
                f'Fold {heldout_fold}: no fit support for '
                f'{CLINICAL_FEATURE_NAMES[feature_index]}.'
            )
        if should_scale:
            imputation[feature_index] = np.float32(np.median(observed))
            mean[feature_index] = np.float32(np.mean(observed))
            raw_std = float(np.std(observed))
            if raw_std < CLINICAL_SCALER_EPSILON:
                constant[feature_index] = 1
            std[feature_index] = np.float32(
                max(raw_std, CLINICAL_SCALER_EPSILON)
            )
        else:
            imputation[feature_index] = 0.0
            mean[feature_index] = 0.0
            std[feature_index] = 1.0

    scaler_path = (
        CLINICAL_SCALER_ROOT / f'fold_{heldout_fold}_clinical_scaler.npz'
    )
    atomic_write_npz(
        scaler_path,
        feature_names=np.asarray(CLINICAL_FEATURE_NAMES, dtype='U'),
        parser_names=np.asarray(CLINICAL_PARSER_NAMES, dtype='U'),
        scale_mask=CLINICAL_SCALE_MASK.astype(np.uint8),
        imputation=imputation,
        mean=mean,
        std=std,
        observed_count=count,
        constant=constant,
        clip_value=np.asarray(CLINICAL_SCALER_CLIP, dtype=np.float32),
        heldout_fold=np.asarray(heldout_fold, dtype=np.int8),
        fit_subject_count=np.asarray(len(fit_subjects), dtype=np.int16),
        heldout_subject_count=np.asarray(len(heldout_subjects), dtype=np.int16),
        fit_subject_hash=np.asarray(subject_hash(fit_subjects)),
        heldout_subject_hash=np.asarray(subject_hash(heldout_subjects)),
        target_arrays_read=np.asarray(False, dtype=np.uint8),
        outcome_workbooks_read=np.asarray(False, dtype=np.uint8),
    )
    scaler_sha256 = file_sha256(scaler_path)
    fold_clinical_scaler_paths[heldout_fold] = str(scaler_path)
    fold_clinical_scaler_hashes[heldout_fold] = scaler_sha256
    clinical_scaler_rows.append({
        'heldout_fold': heldout_fold,
        'fit_subjects': len(fit_subjects),
        'heldout_subjects': len(heldout_subjects),
        'fit_and_heldout_disjoint': not bool(
            fit_subject_set & heldout_subject_set
        ),
        'fit_subjects_are_train_only': fit_subject_set <= set(
            train_df['subject_id'].astype(str)
        ),
        'fit_subject_hash': subject_hash(fit_subjects),
        'heldout_subject_hash': subject_hash(heldout_subjects),
        'minimum_observed_support': int(count.min()),
        'constant_scaled_features': int(
            constant[CLINICAL_SCALE_MASK].sum()
        ),
        'all_statistics_finite': bool(
            np.isfinite(imputation).all()
            and np.isfinite(mean).all()
            and np.isfinite(std).all()
        ),
        'all_stds_positive': bool((std > 0).all()),
        'heldout_values_used_for_scaler': bool(
            fit_subject_set & heldout_subject_set
        ),
        'validation_values_used_for_scaler': bool(
            fit_subject_set & validation_subjects
        ),
        'test_values_used_for_scaler': bool(
            fit_subject_set & test_subjects
        ),
        'target_arrays_read_for_scaler': False,
        'outcome_workbooks_read_for_scaler': False,
        'scaler_path': str(scaler_path),
        'scaler_sha256': scaler_sha256,
    })

clinical_scaler_df = pd.DataFrame(clinical_scaler_rows)
atomic_write_csv(clinical_scaler_df, CLINICAL_SCALER_AUDIT_PATH)


def load_clinical_scaler(heldout_fold):
    path = Path(fold_clinical_scaler_paths[int(heldout_fold)])
    if file_sha256(path) != fold_clinical_scaler_hashes[int(heldout_fold)]:
        raise RuntimeError(f'Fold-{heldout_fold} clinical scaler changed.')
    with np.load(path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if decode_strings(scaler['feature_names']) != CLINICAL_FEATURE_NAMES:
        raise RuntimeError('Clinical scaler feature order changed.')
    if decode_strings(scaler['parser_names']) != CLINICAL_PARSER_NAMES:
        raise RuntimeError('Clinical scaler parser order changed.')
    expected_shapes = {
        'scale_mask': (22,), 'imputation': (22,), 'mean': (22,),
        'std': (22,), 'observed_count': (22,), 'constant': (22,),
    }
    if any(scaler[name].shape != shape for name, shape in expected_shapes.items()):
        raise RuntimeError(f'Fold-{heldout_fold} clinical scaler shape changed.')
    if not np.array_equal(
        np.asarray(scaler['scale_mask']) > 0, CLINICAL_SCALE_MASK
    ):
        raise RuntimeError(f'Fold-{heldout_fold} clinical scale mask changed.')
    if (
        not np.isfinite(scaler['imputation']).all()
        or not np.isfinite(scaler['mean']).all()
        or not np.isfinite(scaler['std']).all()
        or bool((scaler['std'] <= 0).any())
    ):
        raise RuntimeError(f'Fold-{heldout_fold} clinical scaler is invalid.')
    return scaler


def transform_clinical(subject_id, heldout_fold):
    sidecar_path = clinical_sidecar_path_by_subject[subject_id]
    if file_sha256(sidecar_path) != clinical_sidecar_hash_by_subject[subject_id]:
        raise RuntimeError(f'{subject_id}: clinical sidecar changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError(f'{subject_id}: clinical sidecar contains objects.')
        stored_subject = scalar_string(data['subject_id'])
        stored_names = decode_strings(data['feature_names'])
        stored_parsers = decode_strings(data['parser_names'])
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
    if stored_subject != subject_id:
        raise RuntimeError(f'{subject_id}: clinical sidecar subject mismatch.')
    if stored_names != CLINICAL_FEATURE_NAMES:
        raise RuntimeError(f'{subject_id}: clinical feature order changed.')
    if stored_parsers != CLINICAL_PARSER_NAMES:
        raise RuntimeError(f'{subject_id}: clinical parser order changed.')
    if values.shape != (22,) or valid.shape != (22,):
        raise RuntimeError(f'{subject_id}: clinical sidecar shape changed.')
    if not np.isfinite(values).all():
        raise RuntimeError(f'{subject_id}: clinical sidecar is non-finite.')
    scaler = load_clinical_scaler(heldout_fold)
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -float(np.asarray(scaler['clip_value']).reshape(-1)[0]),
        float(np.asarray(scaler['clip_value']).reshape(-1)[0]),
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed,
        (~valid).astype(np.float32),
    ]).astype(np.float32)
    return token




clinical_transform_rows = []
for heldout_fold in range(NUMBER_OF_FOLDS):
    tokens = []
    missingness_matches = []
    for subject_id in sorted(manifest_subjects):
        token = transform_clinical(subject_id, heldout_fold)
        tokens.append(token)
        missingness_matches.append(np.array_equal(
            token[len(CLINICAL_FEATURE_NAMES):],
            (~clinical_valid_by_subject[subject_id]).astype(np.float32),
        ))
    token_matrix = np.stack(tokens)
    clinical_transform_rows.append({
        'heldout_fold': heldout_fold,
        'subjects_transformed': len(token_matrix),
        'token_dimension': int(token_matrix.shape[1]),
        'all_values_finite': bool(np.isfinite(token_matrix).all()),
        'all_missingness_bits_match_sidecars': bool(all(missingness_matches)),
        'minimum_token_value': float(token_matrix.min()),
        'maximum_token_value': float(token_matrix.max()),
        'all_values_within_frozen_clip': bool(
            (token_matrix >= -CLINICAL_SCALER_CLIP).all()
            and (token_matrix <= CLINICAL_SCALER_CLIP).all()
        ),
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
    })
clinical_transform_df = pd.DataFrame(clinical_transform_rows)
atomic_write_csv(clinical_transform_df, CLINICAL_TRANSFORM_AUDIT_PATH)






def load_graph_scaler(heldout_fold):
    path = fold_graph_scaler_paths[int(heldout_fold)]
    with np.load(path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if file_sha256(path) != fold_graph_scaler_hashes[int(heldout_fold)]:
        raise RuntimeError(f'Fold-{heldout_fold} graph scaler changed.')
    expected_territory_names = decode_strings(graph_schema['node_feature_names'])
    scaler_name_checks = {
        'territory': decode_strings(scaler['territory_feature_names'])
        == expected_territory_names,
        'vessel': decode_strings(scaler['vessel_feature_names'])
        == MODEL_VESSEL_FEATURE_NAMES,
        'case': decode_strings(scaler['case_feature_names'])
        == MODEL_CASE_FEATURE_NAMES,
    }
    if not all(scaler_name_checks.values()):
        failed = [name for name, passed in scaler_name_checks.items() if not passed]
        raise RuntimeError(
            f'Fold-{heldout_fold} graph scaler feature names changed: '
            + ', '.join(failed)
        )
    return scaler


def load_multimodal_subject(manifest_row, heldout_fold):
    if 'subject_id' in manifest_row.index:
        subject_id = str(manifest_row['subject_id'])
    elif manifest_row.name is not None:
        subject_id = str(manifest_row.name)
    else:
        raise RuntimeError(
            'Multimodal loader row has neither a subject_id field nor a '
            'subject_id index value.'
        )
    npz_path = Path(str(manifest_row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(manifest_row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(manifest_row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        manifest_row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')

    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union',
            'lvo_proximity', 'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: NPZ missing inputs: ' + ', '.join(missing)
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != len(EXPECTED_INPUT_CHANNELS):
        raise RuntimeError(f'{subject_id}: image tensor must be [15, X, Y, Z].')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image tensor is non-finite.')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input-channel schema changed.')
    if cow_soft_union.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: CoW shape mismatch.')
    if lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: LVO proximity shape mismatch.')
    if not np.isfinite(cow_soft_union).all() or not np.isfinite(
        lvo_proximity
    ).all():
        raise RuntimeError(f'{subject_id}: spatial vascular prior is non-finite.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_features_source = np.asarray(
            data['vessel_features'], dtype=np.float32
        )
        vessel_feature_valid_source = (
            np.asarray(data['vessel_feature_valid']) > 0
        )
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_features_source = np.asarray(
            data['case_features'], dtype=np.float32
        )
    graph_shape_checks = {
        'territory_features': node_features.shape == (30, 65),
        'territory_feature_valid': node_feature_valid.shape == (30, 65),
        'territory_valid': node_valid.shape == (30,),
        'territory_reliability': node_reliability.shape == (30,),
        'vessel_features': vessel_features_source.shape == (13, 16),
        'vessel_feature_valid': vessel_feature_valid_source.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_features': case_features_source.shape == (11,),
    }
    if not all(graph_shape_checks.values()):
        failed = [name for name, passed in graph_shape_checks.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: ' + ', '.join(failed)
        )
    if not all(
        np.isfinite(array).all()
        for array in [
            node_features, node_reliability, vessel_features_source,
            case_features_source,
        ]
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar contains non-finite values.')

    graph_scaler = load_graph_scaler(heldout_fold)
    territory_mean = np.asarray(graph_scaler['territory_mean'], dtype=np.float32)
    territory_std = np.asarray(graph_scaler['territory_std'], dtype=np.float32)
    vessel_source_indices = np.asarray(
        graph_scaler['vessel_source_indices'], dtype=np.int64
    )
    vessel_mean = np.asarray(graph_scaler['vessel_mean'], dtype=np.float32)
    vessel_std = np.asarray(graph_scaler['vessel_std'], dtype=np.float32)
    case_source_indices = np.asarray(
        graph_scaler['case_source_indices'], dtype=np.int64
    )
    case_scale_mask = np.asarray(graph_scaler['case_scale_mask']) > 0
    case_mean = np.asarray(graph_scaler['case_mean'], dtype=np.float32)
    case_std = np.asarray(graph_scaler['case_std'], dtype=np.float32)
    graph_clip = float(np.asarray(graph_scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ], dtype=np.int64)
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ], dtype=np.int64)
    if not np.array_equal(vessel_source_indices, expected_vessel_indices):
        raise RuntimeError(f'{subject_id}: vessel scaler source indices changed.')
    if not np.array_equal(case_source_indices, expected_case_indices):
        raise RuntimeError(f'{subject_id}: case scaler source indices changed.')
    scaler_shape_checks = {
        'territory_mean': territory_mean.shape == (65,),
        'territory_std': territory_std.shape == (65,),
        'vessel_mean': vessel_mean.shape == (15,),
        'vessel_std': vessel_std.shape == (15,),
        'case_scale_mask': case_scale_mask.shape == (8,),
        'case_mean': case_mean.shape == (8,),
        'case_std': case_std.shape == (8,),
    }
    if not all(scaler_shape_checks.values()):
        failed = [name for name, passed in scaler_shape_checks.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph scaler shape mismatch: ' + ', '.join(failed)
        )
    if (
        not math.isfinite(graph_clip)
        or graph_clip <= 0
        or not all(
            np.isfinite(array).all()
            for array in [
                territory_mean, territory_std, vessel_mean, vessel_std,
                case_mean, case_std,
            ]
        )
        or bool((territory_std <= 0).any())
        or bool((vessel_std <= 0).any())
        or bool((case_std <= 0).any())
    ):
        raise RuntimeError(f'{subject_id}: graph scaler statistics are invalid.')

    scaled_territory = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean[None],
        territory_std[None],
        graph_clip,
    )
    selected_vessel = vessel_features_source[:, vessel_source_indices]
    selected_vessel_valid = (
        vessel_feature_valid_source[:, vessel_source_indices]
        & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel,
        selected_vessel_valid,
        vessel_mean[None],
        vessel_std[None],
        graph_clip,
    )
    selected_case = case_features_source[case_source_indices].astype(np.float32)
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
        / case_std[case_scale_mask],
        -graph_clip,
        graph_clip,
    )

    atlas_enabled = as_bool(manifest_row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(manifest_row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region_map = np.asanyarray(atlas_image.dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels are outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_patch = center_crop_or_pad(image, PATCH_SIZE, (1, 2, 3))
    region_patch = center_crop_or_pad(region_map, PATCH_SIZE, (0, 1, 2))
    cow_patch = center_crop_or_pad(cow_soft_union, PATCH_SIZE, (0, 1, 2))
    lvo_patch = center_crop_or_pad(lvo_proximity, PATCH_SIZE, (0, 1, 2))
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch,
        lvo_patch,
    ]).astype(np.float32)

    cow_prior_valid = bool(
        case_features_source[
            SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')
        ] > 0.5
    )
    registration_confidence = float(
        case_features_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()),
        float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)

    occlusion_feature_names = [
        'cta_mean',
        'cta_p90',
        'tmax_mean',
        'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid.append(0.0)

    clinical_token = transform_clinical(subject_id, heldout_fold)
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch.astype(np.float32),
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(occlusion_valid, dtype=np.float32),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(
            clinical_valid_by_subject[subject_id].any(), dtype=bool
        ),
    }
    return output


def arrays_equal(first, second):
    if set(first) != set(second):
        return False
    return all(np.array_equal(first[key], second[key]) for key in first)


def stack_multimodal_batch(samples):
    output = {}
    for key in samples[0]:
        arrays = [np.asarray(sample[key]) for sample in samples]
        output[key] = torch.from_numpy(np.stack(arrays, axis=0))
    return output






audit = availability_by_subject
train_rows = manifest_df[manifest_df['dev_split'] == 'train'].copy()

def choose_subject(mask, excluded=None):
    excluded = set(excluded or [])
    candidates = [
        subject_id for subject_id in audit.index[mask]
        if subject_id in manifest_subjects and subject_id not in excluded
    ]
    candidates = sorted(candidates)
    if not candidates:
        raise RuntimeError('No subject satisfies a required loader test state.')
    return candidates[0]


fold0_fit_subjects = set(
    train_rows.loc[train_rows['development_fold'] != 0, 'subject_id'].astype(str)
)
fold0_heldout_subjects = set(
    train_rows.loc[train_rows['development_fold'] == 0, 'subject_id'].astype(str)
)
complete_mask = (
    (audit['dev_split'].astype(str) == 'train')
    & audit['atlas_enabled'].map(as_bool)
    & audit['lvo_present'].map(as_bool)
    & audit['valid_territory_nodes'].gt(0)
    & audit['valid_vessel_nodes'].gt(0)
)
complete_subjects = sorted(set(audit.index[complete_mask]))
fit_complete = next(
    subject for subject in complete_subjects if subject in fold0_fit_subjects
)
heldout_complete = next(
    subject for subject in complete_subjects if subject in fold0_heldout_subjects
)
selected_subjects = [fit_complete, heldout_complete]
vessel_missing = choose_subject(
    (audit['dev_split'].astype(str) == 'train')
    & audit['valid_vessel_nodes'].eq(0),
    selected_subjects,
)
selected_subjects.append(vessel_missing)
atlas_disabled = choose_subject(
    (audit['dev_split'].astype(str) == 'train')
    & ~audit['atlas_enabled'].map(as_bool),
    selected_subjects,
)
selected_subjects.append(atlas_disabled)
occlusion_missing = choose_subject(
    (audit['dev_split'].astype(str) == 'train')
    & ~audit['lvo_present'].map(as_bool),
    selected_subjects,
)
selected_subjects.append(occlusion_missing)




manifest_by_subject = manifest_df.set_index('subject_id', drop=False)
sample_rows = []
samples = []
for subject_id in selected_subjects:
    row = manifest_by_subject.loc[subject_id]
    sample_first = load_multimodal_subject(row, heldout_fold=0)
    sample_second = load_multimodal_subject(row, heldout_fold=0)
    deterministic = arrays_equal(sample_first, sample_second)
    finite = all(
        np.isfinite(value).all()
        for value in sample_first.values()
        if np.asarray(value).dtype.kind in {'f', 'c'}
    )
    samples.append(sample_first)
    sample_rows.append({
        'subject_id': subject_id,
        'dev_split': str(row['dev_split']),
        'development_fold': int(row['development_fold']),
        'fold0_role': (
            'fit' if subject_id in fold0_fit_subjects else 'heldout'
        ),
        'atlas_enabled': bool(sample_first['atlas_enabled']),
        'valid_territory_nodes': int(sample_first['territory_valid'].sum()),
        'valid_vessel_nodes': int(sample_first['vessel_valid'].sum()),
        'occlusion_available': bool(sample_first['occlusion_available']),
        'clinical_available': bool(sample_first['clinical_available']),
        'clinical_missing_features': int(
            sample_first['clinical_input'][22:].sum()
        ),
        'all_float_arrays_finite': finite,
        'deterministic_repeat_exact': deterministic,
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
    })

loader_case_df = pd.DataFrame(sample_rows)
atomic_write_csv(loader_case_df, LOADER_CASE_AUDIT_PATH)
batch = stack_multimodal_batch(samples)

expected_batch_shapes = {
    'image': (len(samples), 15, *PATCH_SIZE),
    'region_map': (len(samples), *PATCH_SIZE),
    'territory_features': (len(samples), 30, 65),
    'territory_feature_valid': (len(samples), 30, 65),
    'territory_valid': (len(samples), 30),
    'territory_reliability': (len(samples), 30),
    'vessel_features': (len(samples), 13, 15),
    'vessel_feature_valid': (len(samples), 13, 15),
    'vessel_valid': (len(samples), 13),
    'case_features': (len(samples), 8),
    'atlas_enabled': (len(samples),),
    'vascular_image': (len(samples), 5, *PATCH_SIZE),
    'confidence_features': (len(samples), 4),
    'occlusion_values': (len(samples), 7),
    'occlusion_valid': (len(samples), 7),
    'clinical_input': (len(samples), 44),
    'vascular_available': (len(samples),),
    'occlusion_available': (len(samples),),
    'clinical_available': (len(samples),),
}
batch_rows = []
for key, expected_shape in expected_batch_shapes.items():
    observed_shape = tuple(batch[key].shape)
    batch_rows.append({
        'tensor': key,
        'observed_shape': str(observed_shape),
        'expected_shape': str(expected_shape),
        'shape_matches': observed_shape == expected_shape,
        'finite': bool(torch.isfinite(batch[key]).all().item())
        if batch[key].dtype.is_floating_point else True,
    })
loader_batch_df = pd.DataFrame(batch_rows)
atomic_write_csv(loader_batch_df, LOADER_BATCH_AUDIT_PATH)


split_probe_rows = []
for split in ['val', 'test']:
    probe_row = manifest_df[manifest_df['dev_split'] == split].iloc[0]
    probe = load_multimodal_subject(probe_row, heldout_fold=0)
    split_probe_rows.append({
        'split': split,
        'subject_id': str(probe_row['subject_id']),
        'all_float_arrays_finite': all(
            np.isfinite(value).all()
            for value in probe.values()
            if np.asarray(value).dtype.kind in {'f', 'c'}
        ),
        'target_arrays_read': False,
        'outcome_workbooks_read': False,
    })
split_probe_df = pd.DataFrame(split_probe_rows)






clinical_paths = clinical_sidecar_df.set_index('subject_id')[
    'clinical_sidecar_path'
]
clinical_hashes = clinical_sidecar_df.set_index('subject_id')[
    'clinical_sidecar_sha256'
]
candidate_manifest = manifest_df.copy()
candidate_manifest['trial8_clinical_sidecar_path'] = (
    candidate_manifest['subject_id'].map(clinical_paths)
)
candidate_manifest['trial8_clinical_sidecar_sha256'] = (
    candidate_manifest['subject_id'].map(clinical_hashes)
)
candidate_manifest['trial8_clinical_schema_path'] = str(CLINICAL_SCHEMA_PATH)
candidate_manifest['trial8_clinical_schema_sha256'] = clinical_schema_sha256
candidate_manifest['trial8_v2_multimodal_loader_preflight_passed'] = False
candidate_manifest['trial8_model_training_authorized'] = False

clinical_checks = {
    'exactly_148_clinical_sidecars_written': len(clinical_sidecar_df) == EXPECTED_SUBJECTS,
    'all_clinical_sidecars_pickle_free': bool(
        clinical_sidecar_df['sidecar_pickle_free'].all()
    ),
    'all_clinical_sidecars_pass_integrity': bool(
        clinical_sidecar_df['sidecar_integrity_passed'].all()
    ),
    'clinical_sidecars_cover_all_splits': (
        clinical_sidecar_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
    'exactly_one_clinical_parse_error_quarantined': (
        len(clinical_parse_errors) == EXPECTED_CLINICAL_PARSE_ERRORS
        and clinical_parse_errors[0]['subject_id']
        == EXPECTED_CLINICAL_PARSE_ERROR_SUBJECT
        and clinical_parse_errors[0]['feature_name'] == 'hyperlipidemia'
    ),
    'clinical_schema_dimension_is_44': CLINICAL_TOKEN_DIMENSION == 44,
    'clinical_schema_written': CLINICAL_SCHEMA_PATH.exists(),
    'clinical_sidecars_target_free': not bool(
        clinical_sidecar_df['target_arrays_read'].any()
    ),
    'clinical_sidecars_outcome_free': not bool(
        clinical_sidecar_df['outcome_workbooks_read'].any()
    ),
}

scaler_checks = {
    'five_clinical_scalers_written': len(clinical_scaler_df) == NUMBER_OF_FOLDS,
    'clinical_scaler_fit_counts_match': clinical_scaler_df[
        'fit_subjects'
    ].tolist() == [82, 82, 82, 81, 81],
    'clinical_scaler_heldout_counts_match': clinical_scaler_df[
        'heldout_subjects'
    ].tolist() == [20, 20, 20, 21, 21],
    'all_fit_and_heldout_sets_disjoint': bool(
        clinical_scaler_df['fit_and_heldout_disjoint'].all()
    ),
    'all_clinical_scaler_fit_subjects_are_train_only': bool(
        clinical_scaler_df['fit_subjects_are_train_only'].all()
    ),
    'all_clinical_features_have_fold_fit_support': bool(
        clinical_scaler_df['minimum_observed_support'].gt(0).all()
    ),
    'all_clinical_scaler_statistics_finite': bool(
        clinical_scaler_df['all_statistics_finite'].all()
    ),
    'all_clinical_scaler_stds_positive': bool(
        clinical_scaler_df['all_stds_positive'].all()
    ),
    'heldout_values_not_used_for_scalers': not bool(
        clinical_scaler_df['heldout_values_used_for_scaler'].any()
    ),
    'validation_values_not_used_for_scalers': not bool(
        clinical_scaler_df['validation_values_used_for_scaler'].any()
    ),
    'test_values_not_used_for_scalers': not bool(
        clinical_scaler_df['test_values_used_for_scaler'].any()
    ),
    'scaler_fit_target_free': not bool(
        clinical_scaler_df['target_arrays_read_for_scaler'].any()
    ),
    'scaler_fit_outcome_free': not bool(
        clinical_scaler_df['outcome_workbooks_read_for_scaler'].any()
    ),
    'all_148_clinical_sidecars_transform_under_every_fold': bool(
        clinical_transform_df['subjects_transformed'].eq(EXPECTED_SUBJECTS).all()
    ),
    'all_fold_clinical_tokens_have_dimension_44': bool(
        clinical_transform_df['token_dimension'].eq(44).all()
    ),
    'all_fold_clinical_tokens_are_finite': bool(
        clinical_transform_df['all_values_finite'].all()
    ),
    'all_fold_clinical_missingness_bits_match_sidecars': bool(
        clinical_transform_df[
            'all_missingness_bits_match_sidecars'
        ].all()
    ),
    'all_fold_clinical_tokens_respect_frozen_clip': bool(
        clinical_transform_df['all_values_within_frozen_clip'].all()
    ),
}

loader_checks = {
    'real_batch_has_complete_fit_and_heldout_cases': (
        {'fit', 'heldout'} <= set(loader_case_df['fold0_role'])
    ),
    'real_batch_contains_atlas_disabled_case': bool(
        (~loader_case_df['atlas_enabled']).any()
    ),
    'real_batch_contains_vessel_missing_case': bool(
        loader_case_df['valid_vessel_nodes'].eq(0).any()
    ),
    'real_batch_contains_occlusion_missing_case': bool(
        (~loader_case_df['occlusion_available']).any()
    ),
    'all_real_case_arrays_finite': bool(
        loader_case_df['all_float_arrays_finite'].all()
    ),
    'all_real_case_loads_deterministic': bool(
        loader_case_df['deterministic_repeat_exact'].all()
    ),
    'all_batch_shapes_match_v2_forward_contract': bool(
        loader_batch_df['shape_matches'].all()
    ),
    'all_batch_tensors_finite': bool(loader_batch_df['finite'].all()),
    'atlas_disabled_case_has_zero_regions_and_nodes': all(
        int(samples[index]['region_map'].max()) == 0
        and int(samples[index]['territory_valid'].sum()) == 0
        for index, row in loader_case_df.iterrows()
        if not bool(row['atlas_enabled'])
    ),
    'vessel_missing_case_has_zero_vessel_inputs': all(
        int(samples[index]['vessel_valid'].sum()) == 0
        and int(samples[index]['vessel_feature_valid'].sum()) == 0
        and np.count_nonzero(samples[index]['vessel_features']) == 0
        for index, row in loader_case_df.iterrows()
        if int(row['valid_vessel_nodes']) == 0
    ),
    'occlusion_missing_case_is_explicitly_masked': all(
        not bool(samples[index]['occlusion_available'])
        and float(samples[index]['confidence_features'][3]) == 0.0
        for index, row in loader_case_df.iterrows()
        if not bool(row['occlusion_available'])
    ),
    'validation_and_test_inputs_transform_finitely': bool(
        split_probe_df['all_float_arrays_finite'].all()
    ),
    'validation_and_test_targets_not_read': not bool(
        split_probe_df['target_arrays_read'].any()
    ),
    'validation_and_test_outcomes_not_read': not bool(
        split_probe_df['outcome_workbooks_read'].any()
    ),
}

candidate_checks = {
    'candidate_manifest_has_148_subjects': len(candidate_manifest) == EXPECTED_SUBJECTS,
    'candidate_manifest_subjects_unique': (
        candidate_manifest['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'candidate_clinical_paths_complete': candidate_manifest[
        'trial8_clinical_sidecar_path'
    ].notna().all(),
    'candidate_clinical_hashes_complete': candidate_manifest[
        'trial8_clinical_sidecar_sha256'
    ].notna().all(),
    'candidate_training_remains_locked': not bool(candidate_manifest[
        'trial8_model_training_authorized'
    ].map(as_bool).any()),
}

substantive_checks = {
    **upstream_checks,
    **manifest_checks,
    **clinical_checks,
    **scaler_checks,
    **loader_checks,
    **candidate_checks,
    'target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'study_model_not_constructed': True,
    'training_not_performed': True,
    'study_trainable_parameters_are_zero': True,
    'fold0_pilot_training_remains_locked': True,
    'full_fivefold_training_remains_locked': True,
}
substantive_preflight_passed = bool(all(substantive_checks.values()))




candidate_manifest['trial8_v2_multimodal_loader_preflight_passed'] = (
    substantive_preflight_passed
)
atomic_write_csv(candidate_manifest, CANDIDATE_MANIFEST_PATH)
candidate_manifest_sha256 = file_sha256(CANDIDATE_MANIFEST_PATH)
committed_candidate = pd.read_csv(CANDIDATE_MANIFEST_PATH)
commit_checks = {
    'committed_candidate_exists': CANDIDATE_MANIFEST_PATH.exists(),
    'committed_candidate_hash_is_stable': (
        file_sha256(CANDIDATE_MANIFEST_PATH) == candidate_manifest_sha256
    ),
    'committed_candidate_has_148_subjects': (
        len(committed_candidate) == EXPECTED_SUBJECTS
    ),
    'committed_candidate_subjects_unique': (
        committed_candidate['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'committed_candidate_loader_status_matches_substantive_checks': bool(
        committed_candidate[
            'trial8_v2_multimodal_loader_preflight_passed'
        ].map(as_bool).eq(substantive_preflight_passed).all()
    ),
    'committed_candidate_training_is_locked': not bool(
        committed_candidate['trial8_model_training_authorized']
        .map(as_bool).any()
    ),
}
all_checks = {**substantive_checks, **commit_checks}
preflight_passed = bool(all(all_checks.values()))


if not preflight_passed and substantive_preflight_passed:
    candidate_manifest['trial8_v2_multimodal_loader_preflight_passed'] = False
    atomic_write_csv(candidate_manifest, CANDIDATE_MANIFEST_PATH)
    candidate_manifest_sha256 = file_sha256(CANDIDATE_MANIFEST_PATH)

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'preflight_passed': preflight_passed,
    'checks': all_checks,
    'target_policy_manifest_path': str(TARGET_POLICY_MANIFEST_PATH),
    'target_policy_manifest_sha256': target_policy_manifest_sha256,
    'fold_manifest_path': str(FOLD_MANIFEST_PATH),
    'fold_manifest_sha256': fold_manifest_sha256,
    'graph_schema_path': str(GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': graph_schema_sha256,
    'v2_model_preflight_gate_path': str(MODEL_PREFLIGHT_GATE_PATH),
    'v2_architecture_parameters': EXPECTED_V2_PARAMETERS,
    'clinical_schema_path': str(CLINICAL_SCHEMA_PATH),
    'clinical_schema_sha256': clinical_schema_sha256,
    'clinical_sidecar_root': str(CLINICAL_SIDECAR_ROOT),
    'clinical_case_audit_path': str(CLINICAL_CASE_AUDIT_PATH),
    'clinical_scaler_paths': fold_clinical_scaler_paths,
    'clinical_scaler_sha256': fold_clinical_scaler_hashes,
    'clinical_scaler_audit_path': str(CLINICAL_SCALER_AUDIT_PATH),
    'clinical_transform_audit_path': str(CLINICAL_TRANSFORM_AUDIT_PATH),
    'graph_scaler_paths': {
        key: str(value) for key, value in fold_graph_scaler_paths.items()
    },
    'graph_scaler_sha256': fold_graph_scaler_hashes,
    'loader_case_audit_path': str(LOADER_CASE_AUDIT_PATH),
    'loader_batch_audit_path': str(LOADER_BATCH_AUDIT_PATH),
    'candidate_manifest_path': str(CANDIDATE_MANIFEST_PATH),
    'candidate_manifest_sha256': candidate_manifest_sha256,
    'clinical_token_dimension': CLINICAL_TOKEN_DIMENSION,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'study_model_constructed': False,
    'training_performed': False,
    'study_trainable_parameters': 0,
    'fold0_pilot_training_authorized': False,
    'full_fivefold_training_authorized': False,
    'pilot_training_implementation_authorized': preflight_passed,
    'next_block': (
        'build the isolated fold-0 v2 pilot training cell with epoch-level '
        'metrics, early stopping, calibration, and locked test targets'
    ),
}
atomic_write_json(gate, PREFLIGHT_GATE_PATH)

print('\nClinical sidecar summary by split:')
display(clinical_sidecar_df.groupby('dev_split').agg(
    subjects=('subject_id', 'count'),
    mean_present_features=('features_present', 'mean'),
    minimum_present_features=('features_present', 'min'),
    parse_errors=('parse_error_count', 'sum'),
).reset_index())
print('\nFold-specific clinical scaler audit:')
display(clinical_scaler_df)
print('\nFold-specific clinical transform audit:')
display(clinical_transform_df)
print('\nReal multimodal loader cases:')
display(loader_case_df)
print('\nMultimodal batch tensor audit:')
display(loader_batch_df)
print('\nFinal preflight checks:')
display(pd.DataFrame({
    'check': list(all_checks.keys()),
    'passed': list(all_checks.values()),
}))

if not preflight_passed:
    failed = [name for name, passed in all_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 MULTIMODAL LOADER PREFLIGHT FAILED. Do not train. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 V2 MULTIMODAL LOADER AND CLINICAL-SCALER PREFLIGHT PASSED')
print('=' * 80)
print(f'Clinical sidecars:                   {len(clinical_sidecar_df)}')
print(f'Fold-specific clinical scalers:      {len(clinical_scaler_df)}')
print(f'Clinical token dimension:            {CLINICAL_TOKEN_DIMENSION}')
print(f'Quarantined clinical parse errors:   {len(clinical_parse_errors)}')
print(f'Real loader cases checked:           {len(loader_case_df)}')
print('Complete and missing inputs tested:  True')
print('Validation/test inputs transformed:  True')
print('Validation/test targets read:        False')
print('Outcome workbooks read:              False')
print('Study trainable parameters:          0 (loader preflight only)')
print('Model training authorized:           False')
print(f'Pilot implementation allowed:        {preflight_passed}')
print(f'Candidate manifest:                  {CANDIDATE_MANIFEST_PATH}')
print(f'Candidate manifest SHA-256:          {candidate_manifest_sha256}')
print(f'Clinical schema:                     {CLINICAL_SCHEMA_PATH}')
print(f'Preflight gate:                      {PREFLIGHT_GATE_PATH}')
print(
    '\nNext: build the isolated fold-0 v2 pilot training cell with epoch-level '
    'metrics, early stopping, calibration, and locked test targets.'
)


In [ ]:
import sys
import os
import gc
import copy
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')
scipy = ensure_package('scipy')

import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AVAILABILITY_VERSION = 'v1_feature_availability_and_test_time_contract'
PREFLIGHT_VERSION = 'v1_zero_init_gated_fusion_construction'
CODE_REVISION = 'r3_pytorch_module_namespace_safe'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_BASELINE_PARAMETERS = 15_052_628

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

AVAILABILITY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / AVAILABILITY_VERSION
)
AVAILABILITY_GATE_PATH = AVAILABILITY_ROOT / 'feature_availability_gate.json'
INPUT_CONTRACT_PATH = AVAILABILITY_ROOT / 'architecture_input_contract.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = OUTPUT_ROOT / 'architecture_config.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_model_construction_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_model_construction_audit.csv'

SEED = 20260927
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_428_468
PARAMETER_BUDGET_MINIMUM = 15_300_000
PARAMETER_BUDGET_MAXIMUM = 15_650_000

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return to_builtin(value.tolist())
    if isinstance(value, np.generic):
        return to_builtin(value.item())
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    AVAILABILITY_GATE_PATH,
    INPUT_CONTRACT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
availability_gate = json.loads(
    AVAILABILITY_GATE_PATH.read_text(encoding='utf-8')
)
input_contract = json.loads(INPUT_CONTRACT_PATH.read_text(encoding='utf-8'))
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')

if availability_gate.get('feature_availability_audit_passed') is not True:
    raise RuntimeError('Vascular-fusion feature-availability gate is not passed.')
if availability_gate.get('architecture_implementation_authorized') is not True:
    raise RuntimeError('Feature audit does not authorize architecture construction.')
if availability_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Feature audit must leave model training locked.')
if availability_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Feature audit did not remain target-free.')
if availability_gate.get('outcome_workbooks_read') is not False:
    raise RuntimeError('Feature audit read outcome workbooks.')
if availability_gate.get('manifest_sha256') != EXPECTED_TARGET_POLICY_MANIFEST_SHA256:
    raise RuntimeError('Feature audit target-policy manifest hash mismatch.')
if availability_gate.get('clinical_token_input_dimension') != CLINICAL_INPUT_DIM:
    raise RuntimeError('Clinical-token input dimension changed.')
if input_contract.get('architecture_version') != MODEL_VERSION:
    raise RuntimeError('Architecture input-contract version mismatch.')
if input_contract.get('model_training_authorized') is not False:
    raise RuntimeError('Input contract must leave model training locked.')
if input_contract.get('anti_leakage', {}).get('target_arrays_read') is not False:
    raise RuntimeError('Input contract did not remain target-free.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print(f'Trial8 v2 model preflight revision: {CODE_REVISION}')
print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Vascular-fusion availability gate: passed')
print('Target arrays read: False')
print('Outcome workbooks read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

atomic_write_npz(
    SCALER_NPZ_PATH,
    territory_feature_names=np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    territory_mean=territory_mean,
    territory_std=territory_std,
    territory_count=territory_count,
    vessel_feature_names=np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    vessel_source_indices=np.asarray(vessel_source_indices, dtype=np.int16),
    vessel_mean=vessel_mean,
    vessel_std=vessel_std,
    vessel_count=vessel_count,
    case_feature_names=np.asarray(MODEL_CASE_FEATURE_NAMES),
    case_source_indices=np.asarray(case_source_indices, dtype=np.int16),
    case_scale_mask=case_scale_mask.astype(np.uint8),
    case_mean=case_mean,
    case_std=case_std,
    case_count=case_count,
    clip_value=np.asarray(SCALER_CLIP, dtype=np.float32),
    fit_split=np.asarray('train'),
    fit_subject_count=np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    fit_subject_hash=np.asarray(scaler_fit_subject_hash),
    target_arrays_read=np.asarray(False, dtype=np.uint8),
)
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(scaler_record, SCALER_JSON_PATH)

print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    

    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(
                math.atanh(float(fraction_of_bound))
            )


class AsymmetricVascularStream3D(nn.Module):
    

    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        pooled = v5.mean(dim=(2, 3, 4))
        return v3, v4, v5, pooled


class SpatialConfidenceFusion3D(nn.Module):
    

    def __init__(
        self,
        vascular_channels,
        image_channels,
        confidence_dim=4,
        bound=0.1,
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, kernel_size=1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound
        )

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature,
                size=image_feature.shape[2:],
                mode='trilinear',
                align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.value_dim = int(value_dim)
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        safe_values = values * valid
        return self.encoder(torch.cat([safe_values, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    

    def __init__(
        self,
        vascular_dim,
        occlusion_dim,
        clinical_dim,
        output_dim=64,
        confidence_dim=4,
        bound=0.1,
        dropout=0.10,
    ):
        super().__init__()
        source_dim = vascular_dim + occlusion_dim + clinical_dim
        self.source = nn.Sequential(
            nn.Linear(source_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self,
        graph_context,
        vascular_context,
        occlusion_context,
        clinical_context,
        confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context,
            occlusion_context,
            clinical_context,
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
        vascular_channels=(8, 16, 32, 48),
        clinical_input_dim=44,
        clinical_embed_dim=32,
        occlusion_value_dim=7,
        occlusion_embed_dim=32,
        confidence_dim=4,
        fusion_bound=0.1,
        vascular_dropout=0.15,
        clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)

        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        vc1, vc2, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vascular_dim=vc4,
            occlusion_dim=occlusion_embed_dim,
            clinical_dim=clinical_embed_dim,
            output_dim=graph_hidden,
            confidence_dim=confidence_dim,
            bound=fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        modules = [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]
        for module in modules:
            module.activate_for_gradient_test(0.5)

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        vascular_image,
        confidence_features,
        occlusion_values,
        occlusion_valid,
        clinical_input,
        vascular_available,
        occlusion_available,
        clinical_available,
        force_atlas_dropout=None,
        force_vascular_dropout=None,
        force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available,
            self.vascular_dropout,
            force_vascular_dropout,
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available,
            self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available,
            self.clinical_dropout,
            force_clinical_dropout,
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        vascular_gate = effective_vascular.to(image.dtype)
        safe_vascular_image = (
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        v3, v4, v5, vascular_context = self.vascular_stream(
            safe_vascular_image
        )
        vascular_context = (
            vascular_context * vascular_gate[:, None]
        )
        safe_occlusion_valid = (
            occlusion_valid
            * effective_occlusion[:, None].to(occlusion_valid.dtype)
        )
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input)
        clinical_context = (
            clinical_context * effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context,
            vascular_context,
            occlusion_context,
            clinical_context,
            effective_confidence,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')


preflight_row = None
for _, candidate_row in preflight_candidates.iterrows():
    candidate_npz_path = Path(str(candidate_row['vascufate_v2_npz_path']))
    with np.load(candidate_npz_path, allow_pickle=False) as candidate_data:
        candidate_lvo_present = (
            'lvo_present' in candidate_data.files
            and bool(np.asarray(candidate_data['lvo_present']).reshape(-1)[0])
        )
    if candidate_lvo_present:
        preflight_row = candidate_row
        break
if preflight_row is None:
    raise RuntimeError(
        'No atlas-enabled train-only case has a valid LVO source for the '
        'complete v2 branch smoke test.'
    )
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    required_npz_inputs = {
        'image',
        'subject_id',
        'input_channels',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
    }
    missing_npz_inputs = sorted(required_npz_inputs - set(data.files))
    if missing_npz_inputs:
        raise RuntimeError(
            'Preflight source NPZ lacks required input arrays: '
            + ', '.join(missing_npz_inputs)
        )
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
    input_channels = decode_strings(data['input_channels'])
    cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
    lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
    lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')
if input_channels != EXPECTED_INPUT_CHANNELS:
    raise RuntimeError('Preflight input-channel schema changed.')
if cow_soft_union.shape != image.shape[1:]:
    raise RuntimeError('CoW prior shape differs from the image grid.')
if lvo_proximity.shape != image.shape[1:]:
    raise RuntimeError('LVO proximity shape differs from the image grid.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

cow_prior_valid = bool(
    case_features_source[
        SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')
    ] > 0.5
)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))
cow_patch = center_crop_or_pad(
    cow_soft_union, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
lvo_patch = center_crop_or_pad(
    lvo_proximity, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
vascular_patch = np.stack([
    image_patch[input_channels.index('cta')],
    image_patch[input_channels.index('tmax')],
    image_patch[input_channels.index('perfusion_valid')],
    cow_patch,
    lvo_patch,
], axis=0).astype(np.float32)




clinical_input = np.zeros(CLINICAL_INPUT_DIM, dtype=np.float32)
clinical_input[:22] = np.linspace(-0.5, 0.5, 22, dtype=np.float32)
clinical_input[22:] = (np.arange(22) % 3 == 0).astype(np.float32)

occlusion_feature_names = [
    'cta_mean',
    'cta_p90',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
occlusion_values = [float(lvo_present), float(lvo_patch.max())]
occlusion_valid = [1.0, float(lvo_present)]
for feature_name in occlusion_feature_names:
    feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
    valid_nodes = selected_vessel_valid[:, feature_index]
    if bool(valid_nodes.any()):
        occlusion_values.append(float(scaled_vessel[valid_nodes, feature_index].mean()))
        occlusion_valid.append(1.0)
    else:
        occlusion_values.append(0.0)
        occlusion_valid.append(0.0)
occlusion_values = np.asarray(occlusion_values, dtype=np.float32)
occlusion_valid = np.asarray(occlusion_valid, dtype=np.float32)
if occlusion_values.shape != (OCCLUSION_VALUE_DIM,):
    raise RuntimeError('Occlusion-source vector dimension changed.')

confidence_features = np.asarray([
    float(vascular_patch[2].mean()),
    float(cow_prior_valid),
    float(np.clip(selected_case[1], 0.0, 1.0)),
    float(lvo_present),
], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
    'vascular_image': torch.from_numpy(vascular_patch[None]).to(device),
    'confidence_features': torch.from_numpy(
        confidence_features[None]
    ).to(device),
    'occlusion_values': torch.from_numpy(occlusion_values[None]).to(device),
    'occlusion_valid': torch.from_numpy(occlusion_valid[None]).to(device),
    'clinical_input': torch.from_numpy(clinical_input[None]).to(device),
    'vascular_available': torch.ones(1, dtype=torch.bool, device=device),
    'occlusion_available': torch.as_tensor(
        [lvo_present], dtype=torch.bool, device=device
    ),
    'clinical_available': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
vascular_stream_parameters = parameter_count(model.vascular_stream)
spatial_fusion_parameters = parameter_count(
    model.vascular_fusion_quarter
) + parameter_count(model.vascular_fusion_eighth) + parameter_count(
    model.vascular_fusion_bottleneck
)
clinical_parameters = parameter_count(model.clinical_encoder)
occlusion_parameters = parameter_count(model.occlusion_encoder)
context_fusion_parameters = parameter_count(model.context_fusion)
new_fusion_parameters = (
    vascular_stream_parameters
    + spatial_fusion_parameters
    + clinical_parameters
    + occlusion_parameters
    + context_fusion_parameters
)
backbone_parameters = total_parameters - (
    graph_parameters
    + film_parameters
    + head_parameters
    + new_fusion_parameters
)
preserved_v1_parameters = (
    backbone_parameters + graph_parameters + film_parameters + head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'asymmetric vascular stream', 'parameters': vascular_stream_parameters},
    {'component': 'confidence-gated spatial fusion', 'parameters': spatial_fusion_parameters},
    {'component': 'clinical token encoder', 'parameters': clinical_parameters},
    {'component': 'occlusion source encoder', 'parameters': occlusion_parameters},
    {'component': 'gated context fusion', 'parameters': context_fusion_parameters},
    {'component': 'preserved v1 path', 'parameters': preserved_v1_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad():
    normal_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    atlas_dropped_output = model(
        **batch,
        force_atlas_dropout=True,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    new_sources_dropped_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=True,
        force_clinical_dropout=True,
    )

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
    'fused_global_context': (1, GRAPH_HIDDEN),
    'new_source_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(new_sources_dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        atlas_dropped_output['atlas_dropped'].item()
        and not atlas_dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'],
        atlas_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'initial_segmentation_is_new_sources_neutral': torch.allclose(
        normal_output['logits'],
        new_sources_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
    'zero_init_new_fusion_gates_are_zero': all(
        torch.count_nonzero(module.network[-1].weight).item() == 0
        and torch.count_nonzero(module.network[-1].bias).item() == 0
        for module in [
            model.vascular_fusion_quarter.gate,
            model.vascular_fusion_eighth.gate,
            model.vascular_fusion_bottleneck.gate,
            model.context_fusion.gate,
        ]
    ),
    'normal_vascular_source_enabled': bool(
        normal_output['effective_vascular_enabled'].item()
    ),
    'normal_clinical_source_enabled': bool(
        normal_output['effective_clinical_enabled'].item()
    ),
    'forced_new_source_dropout_applied': bool(
        new_sources_dropped_output['vascular_dropped'].item()
        and new_sources_dropped_output['occlusion_dropped'].item()
        and new_sources_dropped_output['clinical_dropped'].item()
    ),
})




model.activate_new_fusion_gates_for_gradient_test()
with torch.no_grad():
    nn.init.normal_(model.global_film.project[-1].weight, mean=0.0, std=1e-3)
    nn.init.zeros_(model.global_film.project[-1].bias)

model.eval()
with torch.no_grad():
    activated_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    vascular_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    occlusion_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=True,
        force_clinical_dropout=False,
    )
    clinical_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=True,
    )

ablation_differences = {
    'vascular_max_abs_logit_change': float(
        (activated_output['logits'] - vascular_ablation_output['logits'])
        .abs().max().item()
    ),
    'occlusion_max_abs_logit_change': float(
        (activated_output['logits'] - occlusion_ablation_output['logits'])
        .abs().max().item()
    ),
    'clinical_max_abs_logit_change': float(
        (activated_output['logits'] - clinical_ablation_output['logits'])
        .abs().max().item()
    ),
}
ablation_checks = {
    'activated_outputs_finite': all(
        torch.isfinite(activated_output[name]).all().item()
        for name in expected_shapes
    ),
    'vascular_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['vascular_max_abs_logit_change'] > 1e-8
    ),
    'occlusion_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['occlusion_max_abs_logit_change'] > 1e-8
    ),
    'clinical_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['clinical_max_abs_logit_change'] > 1e-8
    ),
    'all_activated_gates_remain_bounded': all(
        float(activated_output[name].abs().max().item()) <= FUSION_BOUND + 1e-6
        for name in [
            'context_fusion_gate',
            'vascular_quarter_gate',
            'vascular_eighth_gate',
            'vascular_bottleneck_gate',
        ]
    ),
}

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=False,
):
    gradient_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('vascular_stream', model.vascular_stream),
    ('spatial_fusion', nn.ModuleList([
        model.vascular_fusion_quarter,
        model.vascular_fusion_eighth,
        model.vascular_fusion_bottleneck,
    ])),
    ('clinical_encoder', model.clinical_encoder),
    ('occlusion_encoder', model.occlusion_encoder),
    ('context_fusion', model.context_fusion),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'vascular_stream_has_gradients_after_gate_activation': (
        gradient_norms['vascular_stream'] > 0
    ),
    'spatial_fusion_has_gradients_after_gate_activation': (
        gradient_norms['spatial_fusion'] > 0
    ),
    'clinical_encoder_has_gradients_after_gate_activation': (
        gradient_norms['clinical_encoder'] > 0
    ),
    'occlusion_encoder_has_gradients_after_gate_activation': (
        gradient_norms['occlusion_encoder'] > 0
    ),
    'context_fusion_has_gradients_after_gate_activation': (
        gradient_norms['context_fusion'] > 0
    ),
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'image_channels': int(image.shape[0]),
    'patch_size': PATCH_SIZE,
    'backbone_channels': [32, 64, 128, 224, 320],
    'baseline_architecture_preserved': True,
    'baseline_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    'graph_variant': 'two_layer_relation_aware_gated_GATv2',
    'graph_hidden': GRAPH_HIDDEN,
    'graph_heads': GRAPH_HEADS,
    'graph_layers': GRAPH_LAYERS,
    'relation_bases': RELATION_BASES,
    'graph_dropout': GRAPH_DROPOUT,
    'edge_weight_use': 'log_attention_prior',
    'territory_nodes': NUM_TERRITORY_NODES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'conditioning': (
        'preserved_bounded_zero_initialized_multiscale_FiLM_plus_'
        'confidence_gated_zero_initialized_vascular_context_fusion'
    ),
    'film_bound': FILM_BOUND,
    'fusion_bound': FUSION_BOUND,
    'asymmetric_vascular_stream': {
        'input_channels': [
            'cta', 'tmax', 'perfusion_valid',
            'cow_soft_union', 'lvo_proximity',
        ],
        'channels': VASCULAR_CHANNELS,
        'fusion_scales': ['quarter', 'eighth', 'sixteenth'],
        'initial_gate': 'exact_zero',
    },
    'clinical_token': {
        'input_dimension': CLINICAL_INPUT_DIM,
        'embedding_dimension': CLINICAL_EMBED_DIM,
        'values': 22,
        'paired_missingness_indicators': 22,
        'fold_specific_scaler_required_before_training': True,
    },
    'occlusion_source': {
        'value_dimension': OCCLUSION_VALUE_DIM,
        'embedding_dimension': OCCLUSION_EMBED_DIM,
        'validity_masked': True,
        'uses_tmax_not_ttp': True,
    },
    'confidence_features': [
        'perfusion_valid_fraction',
        'cow_prior_valid',
        'registration_confidence',
        'lvo_present',
    ],
    'staged_neutrality': {
        'new_fusion_gates_exactly_zero_at_initialization': True,
        'controlled_gate_activation_used_only_for_preflight': True,
        'saved_or_trained_weights_created': False,
    },
    'graph_to_logit_residual': False,
    'expert_router': False,
    'deep_supervision': True,
    'territory_auxiliary_head': True,
    'scaler_policy': 'fit_on_train_only_then_freeze',
    'expected_total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
        'asymmetric_vascular_stream': vascular_stream_parameters,
        'spatial_fusion': spatial_fusion_parameters,
        'clinical_encoder': clinical_parameters,
        'occlusion_encoder': occlusion_parameters,
        'context_fusion': context_fusion_parameters,
        'new_fusion_total': new_fusion_parameters,
        'preserved_v1_total': preserved_v1_parameters,
    },
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_sha256': scaler_sha256,
    'feature_availability_gate_path': str(AVAILABILITY_GATE_PATH),
    'architecture_input_contract_path': str(INPUT_CONTRACT_PATH),
    'ablation_differences': ablation_differences,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'model_training_authorized': False,
}
atomic_write_json(architecture_config, ARCHITECTURE_JSON_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **forward_checks,
    **ablation_checks,
    **gradient_checks,
    'feature_availability_audit_passed': (
        availability_gate.get('feature_availability_audit_passed') is True
    ),
    'input_contract_authorizes_architecture_implementation': (
        input_contract.get('architecture_implementation_authorized') is True
    ),
    'preserved_v1_parameter_count_matches': (
        preserved_v1_parameters == EXPECTED_BASELINE_PARAMETERS
    ),
    'parameter_count_in_frozen_efficiency_budget': (
        PARAMETER_BUDGET_MINIMUM
        <= total_parameters
        <= PARAMETER_BUDGET_MAXIMUM
    ),
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'new_fusion_parameter_count_is_bounded': (
        0 < new_fusion_parameters < 600_000
    ),
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'direct_graph_to_logit_residual_disabled': True,
    'expert_routing_disabled': True,
    'training_remained_locked': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'ablation', 'check': name, 'passed': passed}
    for name, passed in ablation_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
 ] + [
    {'category': 'final_gate', 'check': name, 'passed': passed}
    for name, passed in preflight_checks.items()
    if name not in {
        *manifest_checks.keys(),
        *scaler_checks.keys(),
        *forward_checks.keys(),
        *ablation_checks.keys(),
        *gradient_checks.keys(),
    }
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'ablation_differences': ablation_differences,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'controlled_gate_activation_was_diagnostic_only': True,
    'saved_or_trained_weights_created': False,
    'v2_loader_and_clinical_scaler_implementation_authorized': preflight_passed,
    'model_training_authorized': False,
}
atomic_write_json(gate_record, PREFLIGHT_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 MODEL CONSTRUCTION PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'New fusion parameters:              {new_fusion_parameters:,}')
print(f'  Asymmetric vascular stream:       {vascular_stream_parameters:,}')
print(f'  Spatial confidence fusion:        {spatial_fusion_parameters:,}')
print(f'  Clinical encoder:                 {clinical_parameters:,}')
print(f'  Occlusion encoder:                {occlusion_parameters:,}')
print(f'  Context fusion:                   {context_fusion_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('New fusion initially neutral:       True')
print('Controlled ablations responsive:    True')
print('Graph scaler fitted on:             train only (102 subjects)')
print('Clinical scaler fitted:             False (next preflight)')
print('Target arrays read:                 False')
print('Outcome workbooks read:              False')
print('Model training authorized:          False')
print(f'V2 loader implementation allowed:   {preflight_passed}')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print(
    '\nThe standalone block will now verify the completed multimodal-loader '
    'gate and continue into the isolated fold-0 pilot.'
)






PILOT_CODE_REVISION = 'r2_fold0_contract_hardened'
PILOT_VERSION = 'v1_fold0_v2_multimodal_ema_calibrated'
LOADER_PREFLIGHT_VERSION = 'v1_fold_clinical_scalers_multimodal_loader'
FIVEFOLD_PREFLIGHT_VERSION = 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_V2_PARAMETERS = 15_428_468
HELDOUT_FOLD = 0
PILOT_SEED = 20260928
PILOT_PATCH_SIZE = (128, 128, 64)
NUMBER_OF_FOLDS = 5
MINIMUM_POSITIVE_PATCHES_PER_BIN = 20
MAXIMUM_NEGATIVE_CENTER_ATTEMPTS = 512
MAXIMUM_CENTER_RETRIES = 16
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
PILOT_CLINICAL_FEATURE_NAMES = [
    'sex_female',
    'age_years',
    'atrial_fibrillation',
    'hypertension',
    'diabetes',
    'hyperlipidemia',
    'anticoagulation',
    'lipid_lowering_drugs',
    'platelet_aggregation_inhibitors',
    'glucose_mg_dl',
    'leucocytes_g_l',
    'crp_mg_dl',
    'inr',
    'wake_up_stroke',
    'in_house_stroke',
    'external_referral',
    'onset_to_door_hours',
    'alert_to_door_hours',
    'nihss_at_admission',
    'mrs_at_admission',
    'mrs_premorbid',
    'door_to_imaging_hours',
]

MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
LOADER_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / LOADER_PREFLIGHT_VERSION
)
LOADER_GATE_PATH = LOADER_PREFLIGHT_ROOT / 'v2_multimodal_loader_gate.json'
FIVEFOLD_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / FIVEFOLD_PREFLIGHT_VERSION
)
FIVEFOLD_GATE_PATH = (
    FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)
FIVEFOLD_POLICY_PATH = FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_training_policy.json'

PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_START_GATE_PATH = PILOT_ROOT / 'fold0_pilot_start_gate.json'
PILOT_CONFIG_PATH = PILOT_ROOT / 'fold0_pilot_config.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'fold0_training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'fold0_pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'fold0_pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'fold0_epoch1_schedule_audit.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_json_contract_hash(record, excluded_top_level_keys=()):
    excluded = set(excluded_top_level_keys)
    stable_record = {
        key: value for key, value in record.items() if key not in excluded
    }
    payload = json.dumps(
        to_builtin(stable_record),
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


pilot_required_paths = [
    MULTIMODAL_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    LOADER_GATE_PATH,
    FIVEFOLD_GATE_PATH,
    FIVEFOLD_POLICY_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    ARCHITECTURE_JSON_PATH,
    PREFLIGHT_GATE_PATH,
]
pilot_missing_paths = [str(path) for path in pilot_required_paths if not path.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Fold-0 pilot prerequisites are missing:\n' + '\n'.join(pilot_missing_paths)
    )

multimodal_manifest_sha256 = file_sha256(MULTIMODAL_MANIFEST_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)
loader_gate_sha256 = file_sha256(LOADER_GATE_PATH)
fivefold_gate_sha256 = file_sha256(FIVEFOLD_GATE_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)
v2_model_gate_sha256 = file_sha256(PREFLIGHT_GATE_PATH)
architecture_contract_sha256 = pilot_json_contract_hash(
    architecture_config,
    excluded_top_level_keys={'created_at_utc', 'ablation_differences'},
)
if multimodal_manifest_sha256 != EXPECTED_MULTIMODAL_MANIFEST_SHA256:
    raise RuntimeError('Multimodal candidate manifest hash changed.')
if fold_manifest_sha256 != EXPECTED_FOLD_MANIFEST_SHA256:
    raise RuntimeError('Five-fold manifest hash changed.')

loader_gate = json.loads(LOADER_GATE_PATH.read_text(encoding='utf-8'))
fivefold_gate = json.loads(FIVEFOLD_GATE_PATH.read_text(encoding='utf-8'))
fivefold_policy = json.loads(FIVEFOLD_POLICY_PATH.read_text(encoding='utf-8'))
v2_model_gate = json.loads(PREFLIGHT_GATE_PATH.read_text(encoding='utf-8'))

pilot_upstream_checks = {
    'v2_model_preflight_passed': v2_model_gate.get('preflight_passed') is True,
    'v2_model_parameter_count_matches': int(
        v2_model_gate.get('total_parameters', -1)
    ) == EXPECTED_V2_PARAMETERS,
    'v2_model_architecture_hash_matches': (
        v2_model_gate.get('architecture_config_sha256') == architecture_sha256
    ),
    'multimodal_loader_preflight_passed': (
        loader_gate.get('preflight_passed') is True
    ),
    'multimodal_manifest_hash_matches_loader_gate': (
        loader_gate.get('candidate_manifest_sha256')
        == multimodal_manifest_sha256
    ),
    'fold_manifest_hash_matches_loader_gate': (
        loader_gate.get('fold_manifest_sha256') == fold_manifest_sha256
    ),
    'graph_schema_hash_matches_loader_gate': (
        loader_gate.get('graph_schema_sha256') == safe_schema_sha256
    ),
    'pilot_implementation_authorized': (
        loader_gate.get('pilot_training_implementation_authorized') is True
    ),
    'upstream_training_remained_locked': (
        loader_gate.get('fold0_pilot_training_authorized') is False
        and loader_gate.get('full_fivefold_training_authorized') is False
    ),
    'loader_target_free': loader_gate.get('target_arrays_read') is False,
    'loader_outcome_free': loader_gate.get('outcome_workbooks_read') is False,
    'fivefold_loader_loss_sampler_passed': (
        fivefold_gate.get('preflight_passed') is True
        and fivefold_gate.get(
            'baseline_loader_loss_sampler_contract_verified'
        ) is True
    ),
    'fivefold_policy_training_locked': (
        fivefold_gate.get('fold0_pilot_training_authorized') is False
        and fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed_upstream': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed_upstream': fivefold_gate.get('test_targets_read') is False,
    'architecture_still_has_no_expert_router': (
        architecture_config.get('expert_router') is False
    ),
    'architecture_still_has_no_graph_logit_residual': (
        architecture_config.get('graph_to_logit_residual') is False
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [name for name, passed in pilot_upstream_checks.items() if not passed]
    raise RuntimeError('Fold-0 pilot upstream checks failed: ' + ', '.join(failed))

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].astype(str).tolist() != pilot_fold_df[
    'subject_id'
].astype(str).tolist():
    raise RuntimeError('Multimodal and fold manifests have different subjects.')
if 'development_fold' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_fold'] = pilot_fold_df[
        'development_fold'
    ].astype(int)
if 'development_role' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_role'] = pilot_fold_df[
        'development_role'
    ].astype(str)

pilot_required_columns = {
    'subject_id', 'dev_split', 'development_fold', 'vascufate_v2_npz_path',
    'trial8_npz_sha256', 'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
    'trial8_clinical_sidecar_sha256',
    'trial8_v2_multimodal_loader_preflight_passed',
    'trial8_model_training_authorized', 'trial8_supervision_mode',
    'trial8_effective_target_volume_ml',
    'trial8_effective_lesion_volume_bin', 'trial8_sampling_positive',
    'trial8_hard_metric_eligible',
}
pilot_missing_columns = sorted(
    pilot_required_columns - set(pilot_manifest_df.columns)
)
if pilot_missing_columns:
    raise RuntimeError(
        'Multimodal manifest is missing columns: ' + ', '.join(pilot_missing_columns)
    )

development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'].astype(int) != HELDOUT_FOLD
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'].astype(int) == HELDOUT_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()

pilot_manifest_checks = {
    'manifest_has_148_subjects': len(pilot_manifest_df) == 148,
    'manifest_subjects_unique': pilot_manifest_df['subject_id'].nunique() == 148,
    'split_counts_match': pilot_manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'loader_preflight_passed_for_all': bool(
        pilot_manifest_df[
            'trial8_v2_multimodal_loader_preflight_passed'
        ].map(as_bool).all()
    ),
    'manifest_training_remained_locked': not bool(
        pilot_manifest_df['trial8_model_training_authorized'].map(as_bool).any()
    ),
    'fold0_fit_has_82_subjects': len(fold0_fit_df) == 82,
    'fold0_heldout_has_20_subjects': len(fold0_heldout_df) == 20,
    'locked_validation_has_23_subjects': len(locked_validation_df) == 23,
    'sealed_test_has_23_subjects': len(sealed_test_df) == 23,
    'fold0_fit_and_heldout_are_disjoint': not bool(
        set(fold0_fit_df['subject_id']) & set(fold0_heldout_df['subject_id'])
    ),
}
if not all(pilot_manifest_checks.values()):
    failed = [name for name, passed in pilot_manifest_checks.items() if not passed]
    raise RuntimeError('Fold-0 manifest checks failed: ' + ', '.join(failed))

fold0_graph_scaler_path = Path(loader_gate['graph_scaler_paths'][str(HELDOUT_FOLD)])
fold0_graph_scaler_sha256 = loader_gate['graph_scaler_sha256'][
    str(HELDOUT_FOLD)
]
fold0_clinical_scaler_path = Path(
    loader_gate['clinical_scaler_paths'][str(HELDOUT_FOLD)]
)
fold0_clinical_scaler_sha256 = loader_gate['clinical_scaler_sha256'][
    str(HELDOUT_FOLD)
]
for scaler_path, expected_hash, label in [
    (fold0_graph_scaler_path, fold0_graph_scaler_sha256, 'graph'),
    (fold0_clinical_scaler_path, fold0_clinical_scaler_sha256, 'clinical'),
]:
    if not scaler_path.exists():
        raise FileNotFoundError(f'Fold-0 {label} scaler is missing: {scaler_path}')
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-0 {label} scaler hash changed.')

with np.load(fold0_graph_scaler_path, allow_pickle=False) as data:
    fold0_graph_scaler = {name: np.asarray(data[name]) for name in data.files}
with np.load(fold0_clinical_scaler_path, allow_pickle=False) as data:
    fold0_clinical_scaler = {name: np.asarray(data[name]) for name in data.files}

if decode_strings(fold0_graph_scaler['territory_feature_names']) != (
    EXPECTED_NODE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 territory feature order changed.')
if decode_strings(fold0_graph_scaler['vessel_feature_names']) != (
    MODEL_VESSEL_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 vessel feature order changed.')
if decode_strings(fold0_graph_scaler['case_feature_names']) != (
    MODEL_CASE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 case feature order changed.')
if decode_strings(
    fold0_clinical_scaler['feature_names']
) != PILOT_CLINICAL_FEATURE_NAMES:
    raise RuntimeError('Fold-0 clinical feature order changed.')

fold0_fit_subject_hash = pilot_subject_hash(fold0_fit_df['subject_id'])
fold0_heldout_subject_hash = pilot_subject_hash(fold0_heldout_df['subject_id'])
pilot_scaler_contract_checks = {
    'graph_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_graph_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'graph_scaler_fit_count_is_82': int(
        np.asarray(fold0_graph_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'graph_scaler_fit_subject_hash_matches': scalar_string(
        fold0_graph_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'graph_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_graph_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'clinical_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_clinical_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'clinical_scaler_fit_count_is_82': int(
        np.asarray(fold0_clinical_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'clinical_scaler_heldout_count_is_20': int(
        np.asarray(fold0_clinical_scaler['heldout_subject_count']).reshape(-1)[0]
    ) == 20,
    'clinical_scaler_fit_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'clinical_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'graph_scaler_is_target_free': not bool(
        np.asarray(fold0_graph_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_target_free': not bool(
        np.asarray(fold0_clinical_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_outcome_free': not bool(
        np.asarray(
            fold0_clinical_scaler['outcome_workbooks_read']
        ).reshape(-1)[0]
    ),
}
if not all(pilot_scaler_contract_checks.values()):
    failed = [
        name for name, passed in pilot_scaler_contract_checks.items()
        if not passed
    ]
    raise RuntimeError('Fold-0 scaler contract failed: ' + ', '.join(failed))

print('\n' + '=' * 80)
print('TRIAL8 V2 FOLD-0 PILOT CONTRACT PASSED')
print('=' * 80)
print(f'Pilot code revision:                 {PILOT_CODE_REVISION}')
print(f'Fold-0 fit / held-out:               {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
print(f'Locked validation / sealed test:     {len(locked_validation_df)} / {len(sealed_test_df)}')
print('Target access authorized:            development train split only')
print('Locked validation targets read:      False')
print('Test targets read:                   False')


for disposable_name in [
    'model', 'batch', 'normal_output', 'forced_atlas_output',
    'forced_vascular_output', 'forced_occlusion_output',
    'forced_clinical_output', 'gradient_output', 'smoke_loss',
]:
    globals().pop(disposable_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(value) for value in mask.shape),
        'packed': np.packbits(mask.reshape(-1)),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    return np.unpackbits(record['packed'], count=count).reshape(
        record['shape']
    ).astype(bool)


fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')
all_development_row_by_subject = {
    str(row['subject_id']): row for row in development_df.to_dict('records')
}
pilot_mask_records = {}
pilot_shape_by_subject = {}
pilot_target_rows = []
for row in development_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required_target_arrays = {
            'target_binary', 'target_soft', 'binary_loss_weight',
            'crop_mask', 'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required_target_arrays - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        hard_weight = pilot_scalar(data, 'binary_loss_weight')
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
        source_volume_ml = pilot_scalar(data, 'source_target_volume_ml')
    if target_binary.shape != target_soft.shape or target_binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if not np.isfinite(target_soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target arrays are non-finite.')
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        sampling_support = target_binary
        expected_hard_weight = 1.0
    elif supervision_mode == 'soft_only_subvoxel':
        sampling_support = target_soft > 0
        expected_hard_weight = 0.0
    elif supervision_mode == 'true_empty':
        sampling_support = np.zeros_like(target_binary)
        expected_hard_weight = 1.0
    else:
        raise RuntimeError(f'{subject_id}: invalid supervision mode.')
    exclusion_support = target_binary | (target_soft > 0)
    pilot_shape_by_subject[subject_id] = tuple(
        int(value) for value in target_binary.shape
    )
    if not math.isclose(hard_weight, expected_hard_weight):
        raise RuntimeError(f'{subject_id}: hard-loss policy changed.')
    if subject_id in fit_row_by_subject:
        pilot_mask_records[subject_id] = {
            'sampling_support': pilot_pack_mask(sampling_support),
            'exclusion_support': pilot_pack_mask(exclusion_support),
            'crop_mask': pilot_pack_mask(crop_mask),
        }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_fold': int(row['development_fold']),
        'pilot_role': 'fit' if subject_id in fit_row_by_subject else 'heldout',
        'supervision_mode': supervision_mode,
        'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
        'sampling_support_voxels': int(sampling_support.sum()),
        'binary_voxels': int(target_binary.sum()),
        'soft_nonzero_voxels': int((target_soft > 0).sum()),
        'source_target_volume_ml': source_volume_ml,
        'hard_weight': hard_weight,
        'voxel_volume_ml': float(np.prod(spacing) / 1000.0),
        'target_arrays_read_split': 'train',
    })
pilot_target_df = pd.DataFrame(pilot_target_rows)
atomic_write_csv(pilot_target_df, PILOT_TARGET_AUDIT_PATH)
pilot_target_by_subject = pilot_target_df.set_index('subject_id').to_dict('index')

pilot_target_checks = {
    'exactly_102_development_targets_read': len(pilot_target_df) == 102,
    'target_access_roles_are_fit_or_heldout': set(
        pilot_target_df['pilot_role']
    ) == {'fit', 'heldout'},
    'target_access_is_train_split_only': set(
        pilot_target_df['target_arrays_read_split']
    ) == {'train'},
    'fit_sampling_masks_cover_82_subjects': len(pilot_mask_records) == 82,
    'soft_only_hard_loss_disabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'] == 'soft_only_subvoxel',
            'hard_weight',
        ].eq(0.0).all()
    ),
    'hard_and_empty_hard_loss_enabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'].isin(
                ['hard_and_soft', 'true_empty']
            ),
            'hard_weight',
        ].eq(1.0).all()
    ),
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
}
if not all(pilot_target_checks.values()):
    failed = [name for name, passed in pilot_target_checks.items() if not passed]
    raise RuntimeError('Fold-0 target checks failed: ' + ', '.join(failed))

territory_edge_index_numpy = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type_numpy = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)
PILOT_RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def pilot_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, PILOT_PATCH_SIZE):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def pilot_extract_patch(array, slices, spatial_axes):
    source = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source[axis] = spatial_slice
    cropped = array[tuple(source)]
    return center_crop_or_pad(cropped, PILOT_PATCH_SIZE, spatial_axes)


region_cache = {}
region_cache_order = []
REGION_CACHE_LIMIT = 24


def pilot_load_region_map(subject_id):
    if subject_id in region_cache:
        return region_cache[subject_id]
    row = all_development_row_by_subject[subject_id]
    shape = pilot_shape_by_subject[subject_id]
    if not as_bool(row['trial8_atlas_enabled']):
        region = np.zeros(shape, dtype=np.uint8)
    else:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region = np.asanyarray(atlas_image.dataobj)
        if region.ndim == 4 and region.shape[-1] == 1:
            region = region[..., 0]
        region = np.rint(region).astype(np.uint8)
        if region.shape != shape:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region.min() < 0 or region.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    region_cache[subject_id] = region
    region_cache_order.append(subject_id)
    if len(region_cache_order) > REGION_CACHE_LIMIT:
        evicted = region_cache_order.pop(0)
        region_cache.pop(evicted, None)
    return region


def pilot_topology_candidates(support, region_map):
    lesion_labels = set(
        int(value) for value in np.unique(region_map[support])
        if 1 <= int(value) <= 30
    )
    relation_priority = {2: 0, 1: 1, 3: 2}
    relation_by_label = {}
    for edge_index in range(territory_edge_index_numpy.shape[1]):
        source = int(territory_edge_index_numpy[0, edge_index]) + 1
        destination = int(territory_edge_index_numpy[1, edge_index]) + 1
        relation = int(territory_edge_type_numpy[edge_index])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in relation_priority
        ):
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    labels = sorted(
        relation_by_label,
        key=lambda label: (relation_priority[relation_by_label[label]], label),
    )
    return labels, relation_by_label


def pilot_sample_coordinates(mask, rng):
    flat = np.flatnonzero(mask)
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > MAXIMUM_NEGATIVE_CENTER_ATTEMPTS:
        flat = rng.choice(
            flat, size=MAXIMUM_NEGATIVE_CENTER_ATTEMPTS, replace=False
        )
    else:
        flat = flat.copy()
        rng.shuffle(flat)
    return np.column_stack(np.unravel_index(flat, mask.shape)).astype(np.int64)


def pilot_zero_support_center(exclusion, candidate, rng):
    for center in pilot_sample_coordinates(candidate, rng):
        slices = pilot_patch_slices(exclusion.shape, center)
        if not exclusion[slices].any():
            return center, slices
    return None, None


def pilot_positive_center(subject_id, rng, epoch):
    support = pilot_unpack_mask(
        pilot_mask_records[subject_id]['sampling_support']
    )
    if not support.any():
        raise RuntimeError(f'{subject_id}: positive schedule has no support.')
    if pilot_target_by_subject[subject_id]['supervision_mode'] == (
        'soft_only_subvoxel'
    ):
        labels, component_count = ndimage.label(
            support, structure=np.ones((3, 3, 3), dtype=np.uint8)
        )
        selected_component = 1 + ((epoch - 1) % component_count)
        coordinates = np.argwhere(labels == selected_component)
    else:
        coordinates = np.argwhere(support)
    seed = coordinates[int(rng.integers(0, len(coordinates)))].astype(np.int64)
    for _ in range(MAXIMUM_CENTER_RETRIES):
        jitter = np.asarray([
            int(rng.integers(-12, 13)),
            int(rng.integers(-12, 13)),
            int(rng.integers(-4, 5)),
        ])
        center = np.clip(seed + jitter, 0, np.asarray(support.shape) - 1)
        slices = pilot_patch_slices(support.shape, center)
        if support[slices].any():
            return center, slices
    slices = pilot_patch_slices(support.shape, seed)
    if not support[slices].any():
        raise RuntimeError(f'{subject_id}: failed to retain positive support.')
    return seed, slices


def pilot_balanced_positive_schedule(rng):
    schedule = []
    for lesion_bin in POSITIVE_BINS:
        pool = fold0_fit_df.loc[
            (fold0_fit_df['trial8_effective_lesion_volume_bin'] == lesion_bin)
            & fold0_fit_df['trial8_sampling_positive'].map(as_bool),
            'subject_id',
        ].astype(str).tolist()
        if not pool:
            raise RuntimeError(f'Fold-0 positive bin {lesion_bin} is empty.')
        target_count = max(MINIMUM_POSITIVE_PATCHES_PER_BIN, len(pool))
        selected = pool.copy()
        hard_pool = [
            subject_id for subject_id in pool
            if pilot_target_by_subject[subject_id]['supervision_mode']
            == 'hard_and_soft'
        ]
        rng.shuffle(hard_pool)
        if not hard_pool and len(selected) < target_count:
            raise RuntimeError(f'No hard-target repeats for {lesion_bin}.')
        repeat_index = 0
        while len(selected) < target_count:
            selected.append(hard_pool[repeat_index % len(hard_pool)])
            repeat_index += 1
        rng.shuffle(selected)
        schedule.extend(selected)
    rng.shuffle(schedule)
    return schedule


def pilot_build_epoch_schedule(epoch):
    rng = np.random.default_rng(PILOT_SEED + epoch * 1_000_003)
    positive_subjects = pilot_balanced_positive_schedule(rng)
    positive_count = len(positive_subjects)
    topology_count = int(math.ceil(positive_count / 4.0))
    brain_count = int(math.floor(positive_count / 4.0))
    topology_pool = fold0_fit_df.loc[
        fold0_fit_df['trial8_atlas_enabled'].map(as_bool)
        & fold0_fit_df['trial8_sampling_positive'].map(as_bool)
        & fold0_fit_df['trial8_supervision_mode'].eq('hard_and_soft'),
        'subject_id',
    ].astype(str).tolist()
    brain_pool = fold0_fit_df.loc[
        ~fold0_fit_df['trial8_supervision_mode'].eq('soft_only_subvoxel'),
        'subject_id',
    ].astype(str).tolist()
    entries = []
    for subject_id in positive_subjects:
        center, _ = pilot_positive_center(subject_id, rng, epoch)
        entries.append({
            'sample_type': 'positive',
            'subject_id': subject_id,
            'center': center,
            'topology_relation': '',
        })
    for _ in range(topology_count):
        selected = None
        start = int(rng.integers(0, len(topology_pool)))
        for attempt in range(len(topology_pool)):
            subject_id = topology_pool[(start + attempt) % len(topology_pool)]
            support = pilot_unpack_mask(
                pilot_mask_records[subject_id]['sampling_support']
            )
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            region = pilot_load_region_map(subject_id)
            labels, relation_by_label = pilot_topology_candidates(support, region)
            for relation in [2, 1, 3]:
                relation_labels = [
                    label for label in labels
                    if relation_by_label[label] == relation
                ]
                if not relation_labels:
                    continue
                center, _ = pilot_zero_support_center(
                    exclusion, np.isin(region, relation_labels) & crop_mask, rng
                )
                if center is not None:
                    selected = {
                        'sample_type': 'topology_negative',
                        'subject_id': subject_id,
                        'center': center,
                        'topology_relation': PILOT_RELATION_NAME[relation],
                    }
                    break
            if selected is not None:
                break
        if selected is None:
            raise RuntimeError('Could not construct a topology-negative patch.')
        entries.append(selected)
    for _ in range(brain_count):
        selected = None
        start = int(rng.integers(0, len(brain_pool)))
        for attempt in range(len(brain_pool)):
            subject_id = brain_pool[(start + attempt) % len(brain_pool)]
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            center, _ = pilot_zero_support_center(exclusion, crop_mask, rng)
            if center is not None:
                selected = {
                    'sample_type': 'brain_negative',
                    'subject_id': subject_id,
                    'center': center,
                    'topology_relation': '',
                }
                break
        if selected is None:
            raise RuntimeError('Could not construct a brain-negative patch.')
        entries.append(selected)
    rng.shuffle(entries)
    return entries






graph_input_cache = {}
clinical_input_cache = {}


def pilot_load_graph_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in graph_input_cache:
        return graph_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: graph sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    territory_mean_local = np.asarray(
        fold0_graph_scaler['territory_mean'], dtype=np.float32
    )
    territory_std_local = np.asarray(
        fold0_graph_scaler['territory_std'], dtype=np.float32
    )
    vessel_indices = np.asarray(
        fold0_graph_scaler['vessel_source_indices'], dtype=np.int64
    )
    case_indices = np.asarray(
        fold0_graph_scaler['case_source_indices'], dtype=np.int64
    )
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_indices] & vessel_presence[:, None]
    )
    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean_local[None],
        territory_std_local[None],
    )
    vessel_scaled = masked_scale(
        vessel_source[:, vessel_indices],
        vessel_feature_valid,
        np.asarray(fold0_graph_scaler['vessel_mean'], dtype=np.float32)[None],
        np.asarray(fold0_graph_scaler['vessel_std'], dtype=np.float32)[None],
    )
    case_selected = case_source[case_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scale_mask_local = np.asarray(
        fold0_graph_scaler['case_scale_mask']
    ) > 0
    case_mean_local = np.asarray(
        fold0_graph_scaler['case_mean'], dtype=np.float32
    )
    case_std_local = np.asarray(
        fold0_graph_scaler['case_std'], dtype=np.float32
    )
    case_scaled[case_scale_mask_local] = np.clip(
        (
            case_scaled[case_scale_mask_local]
            - case_mean_local[case_scale_mask_local]
        ) / case_std_local[case_scale_mask_local],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean',
        'minimum_lvo_distance_mm', 'lvo_overlap_fraction',
    ]
    occlusion_graph_values = []
    occlusion_graph_valid = []
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = vessel_feature_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_graph_values.append(float(
                vessel_scaled[valid_nodes, feature_index].mean()
            ))
            occlusion_graph_valid.append(1.0)
        else:
            occlusion_graph_values.append(0.0)
            occlusion_graph_valid.append(0.0)
    output = {
        'territory_features': territory_scaled.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': vessel_scaled.astype(np.float32),
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence.astype(bool),
        'case_features': case_scaled.astype(np.float32),
        'cow_prior_valid': bool(
            case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
        ),
        'registration_confidence': float(
            case_source[
                SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
            ]
        ),
        'occlusion_graph_values': np.asarray(
            occlusion_graph_values, dtype=np.float32
        ),
        'occlusion_graph_valid': np.asarray(
            occlusion_graph_valid, dtype=np.float32
        ),
    }
    if not all(
        np.isfinite(value).all()
        for value in output.values()
        if np.asarray(value).dtype.kind in {'f', 'c'}
    ):
        raise RuntimeError(f'{subject_id}: scaled graph input is non-finite.')
    graph_input_cache[subject_id] = output
    return output


def pilot_load_clinical_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in clinical_input_cache:
        return clinical_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_clinical_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: clinical sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        names = decode_strings(data['feature_names'])
    scaler_names = decode_strings(fold0_clinical_scaler['feature_names'])
    if names != scaler_names or values.shape != (22,) or valid.shape != (22,):
        raise RuntimeError(f'{subject_id}: clinical schema changed.')
    imputed = values.copy()
    imputation = np.asarray(
        fold0_clinical_scaler['imputation'], dtype=np.float32
    )
    imputed[~valid] = imputation[~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(fold0_clinical_scaler['scale_mask']) > 0
    clinical_mean = np.asarray(fold0_clinical_scaler['mean'], dtype=np.float32)
    clinical_std = np.asarray(fold0_clinical_scaler['std'], dtype=np.float32)
    clinical_clip = float(
        np.asarray(fold0_clinical_scaler['clip_value']).reshape(-1)[0]
    )
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - clinical_mean[scale_mask])
        / clinical_std[scale_mask],
        -clinical_clip,
        clinical_clip,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    output = {
        'clinical_input': token,
        'clinical_available': np.asarray(valid.any(), dtype=bool),
    }
    clinical_input_cache[subject_id] = output
    return output


def pilot_model_input_from_patches(
    row,
    image_patch,
    region_patch,
    cow_patch,
    lvo_patch,
    lvo_present,
    apply_graph_mask=False,
    rng=None,
):
    graph = {
        key: np.asarray(value).copy()
        for key, value in pilot_load_graph_input(row).items()
    }
    clinical = pilot_load_clinical_input(row)
    if apply_graph_mask and rng is not None:
        mask = (
            rng.random(graph['territory_features'].shape) < 0.05
        ) & (graph['territory_feature_valid'] > 0)
        graph['territory_features'][mask] = 0.0
        graph['territory_feature_valid'][mask] = 0.0
    input_channels = EXPECTED_INPUT_CHANNELS
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch,
        lvo_patch,
    ]).astype(np.float32)
    confidence = np.asarray([
        float(vascular_patch[2].mean()),
        float(graph['cow_prior_valid']),
        float(np.clip(graph['registration_confidence'], 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_values = np.concatenate([
        np.asarray([float(lvo_present), float(lvo_patch.max())], dtype=np.float32),
        graph['occlusion_graph_values'],
    ]).astype(np.float32)
    occlusion_valid = np.concatenate([
        np.asarray([1.0, float(lvo_present)], dtype=np.float32),
        graph['occlusion_graph_valid'],
    ]).astype(np.float32)
    return {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': graph['territory_features'].astype(np.float32),
        'territory_feature_valid': graph[
            'territory_feature_valid'
        ].astype(np.float32),
        'territory_valid': graph['territory_valid'].astype(bool),
        'territory_reliability': graph['territory_reliability'].astype(np.float32),
        'vessel_features': graph['vessel_features'].astype(np.float32),
        'vessel_feature_valid': graph['vessel_feature_valid'].astype(np.float32),
        'vessel_valid': graph['vessel_valid'].astype(bool),
        'case_features': graph['case_features'].astype(np.float32),
        'atlas_enabled': np.asarray(
            as_bool(row['trial8_atlas_enabled']), dtype=bool
        ),
        'vascular_image': vascular_patch,
        'confidence_features': confidence,
        'occlusion_values': occlusion_values,
        'occlusion_valid': occlusion_valid,
        'clinical_input': clinical['clinical_input'].astype(np.float32),
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': clinical['clinical_available'],
    }


def pilot_intensity_augment(image, rng):
    output = np.asarray(image, dtype=np.float32).copy()
    intensity_channels = min(13, output.shape[0])
    gains = rng.uniform(0.92, 1.08, size=(intensity_channels, 1, 1, 1))
    offsets = rng.uniform(-0.02, 0.02, size=(intensity_channels, 1, 1, 1))
    output[:intensity_channels] = output[:intensity_channels] * gains + offsets
    if rng.random() < 0.50:
        noise_std = float(rng.uniform(0.0, 0.015))
        output[:intensity_channels] += rng.normal(
            0.0, noise_std, size=output[:intensity_channels].shape
        ).astype(np.float32)
    if rng.random() < 0.40:
        gamma = float(rng.uniform(0.90, 1.10))
        nonnegative_end = min(9, intensity_channels)
        output[:nonnegative_end] = np.power(
            np.clip(output[:nonnegative_end], 0.0, 1.0), gamma
        )
    output[:min(9, intensity_channels)] = np.clip(
        output[:min(9, intensity_channels)], 0.0, 1.0
    )
    if intensity_channels > 9:
        output[9:intensity_channels] = np.clip(
            output[9:intensity_channels], -1.0, 1.0
        )
    return output.astype(np.float32)


class Trial8Fold0PatchDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.counts = {}
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)
        self.counts = {
            name: sum(entry['sample_type'] == name for entry in self.schedule)
            for name in ['positive', 'topology_negative', 'brain_negative']
        }

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        with np.load(npz_path, allow_pickle=False) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            input_channels = decode_strings(data['input_channels'])
            target_binary = np.asarray(data['target_binary'], dtype=np.float32)
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
            lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
            lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            hard_weight = pilot_scalar(data, 'binary_loss_weight')
        if input_channels != EXPECTED_INPUT_CHANNELS:
            raise RuntimeError(f'{subject_id}: image channel order changed.')
        slices = pilot_patch_slices(target_binary.shape, entry['center'])
        image_patch = pilot_extract_patch(image, slices, (1, 2, 3))
        binary_patch = pilot_extract_patch(target_binary, slices, (0, 1, 2))
        soft_patch = pilot_extract_patch(target_soft, slices, (0, 1, 2))
        cow_patch = pilot_extract_patch(cow, slices, (0, 1, 2))
        lvo_patch = pilot_extract_patch(lvo, slices, (0, 1, 2))
        region_patch = pilot_extract_patch(
            pilot_load_region_map(subject_id), slices, (0, 1, 2)
        )
        if entry['sample_type'] == 'positive' and not bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: positive patch lost target support.')
        if entry['sample_type'] != 'positive' and bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: negative patch contains target support.')
        image_patch = pilot_intensity_augment(image_patch, rng)
        model_inputs = pilot_model_input_from_patches(
            row,
            image_patch,
            region_patch,
            cow_patch,
            lvo_patch,
            lvo_present,
            apply_graph_mask=True,
            rng=rng,
        )
        model_inputs.update({
            'target_binary': binary_patch[None].astype(np.float32),
            'target_soft': soft_patch[None].astype(np.float32),
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'supervision_mode': str(
                pilot_target_by_subject[subject_id]['supervision_mode']
            ),
        })
        return model_inputs


MODEL_INPUT_KEYS = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available',
]


def pilot_to_device(cpu_batch):
    output = {}
    for key, value in cpu_batch.items():
        if torch.is_tensor(value):
            output[key] = value.to(device, non_blocking=True)
        else:
            output[key] = value
    return output


def pilot_spatial_augment(batch):
    batch_size = int(batch['image'].shape[0])
    if float(torch.rand(1).item()) >= 0.60:
        batch['spatially_augmented_fraction'] = torch.tensor(
            0.0, device=batch['image'].device
        )
        return batch
    positive = [str(value) == 'positive' for value in batch['sample_type']]
    batch_device = batch['image'].device
    for _ in range(4):
        angles = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * math.radians(8.0)
        scales = 0.92 + torch.rand(batch_size, device=batch_device) * 0.16
        cosine = torch.cos(angles) * scales
        sine = torch.sin(angles) * scales
        theta = torch.zeros(
            batch_size, 3, 4, device=batch_device, dtype=torch.float32
        )
        theta[:, 0, 0] = cosine
        theta[:, 0, 1] = -sine
        theta[:, 1, 0] = sine
        theta[:, 1, 1] = cosine
        theta[:, 2, 2] = scales
        theta[:, 0, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        theta[:, 1, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        theta[:, 2, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.025
        grid = F.affine_grid(theta, batch['image'].shape, align_corners=False)

        transformed = dict(batch)
        transformed_image = torch.empty_like(batch['image'])
        transformed_image[:, :13] = F.grid_sample(
            batch['image'][:, :13], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed_image[:, 13:] = F.grid_sample(
            batch['image'][:, 13:], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['image'] = transformed_image
        transformed_vascular = torch.empty_like(batch['vascular_image'])
        continuous_indices = [0, 1, 3, 4]
        transformed_vascular[:, continuous_indices] = F.grid_sample(
            batch['vascular_image'][:, continuous_indices], grid,
            mode='bilinear', padding_mode='zeros', align_corners=False,
        )
        transformed_vascular[:, 2:3] = F.grid_sample(
            batch['vascular_image'][:, 2:3], grid,
            mode='nearest', padding_mode='zeros', align_corners=False,
        )
        transformed['vascular_image'] = transformed_vascular
        transformed['target_soft'] = F.grid_sample(
            batch['target_soft'], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed['target_binary'] = F.grid_sample(
            batch['target_binary'], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['region_map'] = F.grid_sample(
            batch['region_map'][:, None].float(), grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )[:, 0].long()
        positive_support_retained = all(
            (not is_positive) or bool(
                (transformed['target_soft'][batch_index] > 0).any().item()
            )
            for batch_index, is_positive in enumerate(positive)
        )
        if not positive_support_retained:
            continue
        transformed['confidence_features'] = batch[
            'confidence_features'
        ].clone()
        transformed['confidence_features'][:, 0] = transformed_vascular[
            :, 2
        ].float().mean(dim=(1, 2, 3))
        transformed['occlusion_values'] = batch['occlusion_values'].clone()
        transformed['occlusion_values'][:, 1] = transformed_vascular[
            :, 4
        ].float().amax(dim=(1, 2, 3))
        transformed['spatially_augmented_fraction'] = torch.tensor(
            1.0, device=batch_device
        )
        return transformed
    batch['spatially_augmented_fraction'] = torch.tensor(
        0.0, device=batch_device
    )
    return batch






def pilot_soft_dice_per_case(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_soft_focal_bce_per_case(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, raw.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_per_case(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error_floor = 1e-6
    return (
        (1.0 - tversky).clamp_min(error_floor).pow(gamma)
        - float(error_floor ** gamma)
    )


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    dice = pilot_soft_dice_per_case(logits, soft_target)
    focal_bce = pilot_soft_focal_bce_per_case(logits, soft_target)
    hard_tversky = pilot_focal_tversky_per_case(logits, binary_target)
    hard_contribution = 0.20 * hard_tversky * hard_weight.reshape(-1)
    total = 0.50 * dice + 0.30 * focal_bce + hard_contribution
    return total.mean(), {
        'soft_dice_loss': dice.mean(),
        'soft_focal_bce_loss': focal_bce.mean(),
        'hard_focal_tversky_loss': hard_tversky.mean(),
        'hard_contribution': hard_contribution.mean(),
    }


def pilot_node_loss(output, batch):
    node_logits = output['territory_node_logits'].float()
    targets = torch.zeros_like(node_logits)
    valid = (
        batch['territory_valid'].bool()
        & batch['atlas_enabled'][:, None].bool()
        & (batch['hard_weight'][:, None] > 0)
    )
    for batch_index in range(node_logits.shape[0]):
        labels = batch['region_map'][batch_index]
        lesion = batch['target_binary'][batch_index, 0] > 0.5
        if bool(lesion.any().item()):
            present_labels = torch.unique(labels[lesion])
            present_labels = present_labels[
                (present_labels >= 1) & (present_labels <= 30)
            ]
            if len(present_labels) > 0:
                targets[batch_index, present_labels.long() - 1] = 1.0
    if not bool(valid.any().item()):
        return node_logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        node_logits,
        targets,
        reduction='none',
        pos_weight=torch.full(
            (30,), 4.0, device=node_logits.device, dtype=node_logits.dtype
        ),
    )
    return raw[valid].mean()


def pilot_compute_training_loss(study_model, cpu_batch):
    batch = pilot_to_device(cpu_batch)
    batch = pilot_spatial_augment(batch)
    output = study_model(**{key: batch[key] for key in MODEL_INPUT_KEYS})
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    auxiliary_eligible = (batch['hard_weight'] > 0).float()
    half_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], size=output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'],
        size=output['auxiliary_quarter_logits'].shape[2:], mode='nearest',
    )
    half_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_half_logits'], half_binary
        ) * batch['hard_weight']
    )
    quarter_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_quarter_logits'], quarter_binary
        ) * batch['hard_weight']
    )
    denominator = auxiliary_eligible.sum().clamp_min(1.0)
    half_loss = (half_per_case * auxiliary_eligible).sum() / denominator
    quarter_loss = (quarter_per_case * auxiliary_eligible).sum() / denominator
    node_loss = pilot_node_loss(output, batch)
    total = main_loss + 0.15 * half_loss + 0.10 * quarter_loss + 0.02 * node_loss
    parts = {
        'total_loss': total,
        'main_loss': main_loss,
        **main_parts,
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
        'vascular_retained_fraction': output[
            'effective_vascular_enabled'
        ].float().mean(),
        'occlusion_retained_fraction': output[
            'effective_occlusion_enabled'
        ].float().mean(),
        'clinical_retained_fraction': output[
            'effective_clinical_enabled'
        ].float().mean(),
        'spatially_augmented_fraction': batch[
            'spatially_augmented_fraction'
        ],
    }
    return total, parts


def pilot_sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_extract_window(array, starts, spatial_axes):
    output_shape = list(array.shape)
    for axis, patch in zip(spatial_axes, PILOT_PATCH_SIZE):
        output_shape[axis] = patch
    output = np.zeros(output_shape, dtype=array.dtype)
    source = [slice(None)] * array.ndim
    destination = [slice(None)] * array.ndim
    source_spatial = []
    destination_spatial = []
    for axis, start, patch in zip(spatial_axes, starts, PILOT_PATCH_SIZE):
        stop = min(start + patch, array.shape[axis])
        copied = stop - start
        destination_start = max((patch - copied) // 2, 0)
        source[axis] = slice(start, stop)
        destination[axis] = slice(destination_start, destination_start + copied)
        source_spatial.append(slice(start, stop))
        destination_spatial.append(
            slice(destination_start, destination_start + copied)
        )
    output[tuple(destination)] = array[tuple(source)]
    return output, tuple(source_spatial), tuple(destination_spatial)


def pilot_gaussian_weight():
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in PILOT_PATCH_SIZE
    ]
    grids = np.meshgrid(*coordinates, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    weight = np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32)
    return np.maximum(weight, 0.05)


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


@torch.no_grad()
def pilot_full_volume_probability(study_model, row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: validation channel order changed.')
    region = pilot_load_region_map(subject_id)
    spatial_shape = image.shape[1:]
    probability_sum = np.zeros(spatial_shape, dtype=np.float32)
    weight_sum = np.zeros(spatial_shape, dtype=np.float32)
    window_records = []
    for x in pilot_sliding_starts(spatial_shape[0], PILOT_PATCH_SIZE[0]):
        for y in pilot_sliding_starts(spatial_shape[1], PILOT_PATCH_SIZE[1]):
            for z in pilot_sliding_starts(spatial_shape[2], PILOT_PATCH_SIZE[2]):
                window_records.append((x, y, z))
    study_model.eval()
    for start_index in range(0, len(window_records), 2):
        chunk = window_records[start_index:start_index + 2]
        numpy_inputs = []
        mappings = []
        for starts in chunk:
            image_patch, source_slices, destination_slices = pilot_extract_window(
                image, starts, (1, 2, 3)
            )
            region_patch, _, _ = pilot_extract_window(region, starts, (0, 1, 2))
            cow_patch, _, _ = pilot_extract_window(cow, starts, (0, 1, 2))
            lvo_patch, _, _ = pilot_extract_window(lvo, starts, (0, 1, 2))
            numpy_inputs.append(pilot_model_input_from_patches(
                row, image_patch, region_patch, cow_patch, lvo_patch,
                lvo_present, apply_graph_mask=False,
            ))
            mappings.append((source_slices, destination_slices))
        model_batch = {
            key: torch.from_numpy(np.stack([
                np.asarray(item[key]) for item in numpy_inputs
            ])).to(device, non_blocking=True)
            for key in MODEL_INPUT_KEYS
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            output = study_model(
                **model_batch,
                force_atlas_dropout=False,
                force_vascular_dropout=False,
                force_occlusion_dropout=False,
                force_clinical_dropout=False,
            )
        probabilities = torch.sigmoid(output['logits'].float())[
            :, 0
        ].cpu().numpy()
        for local_index, probability_patch in enumerate(probabilities):
            source_slices, destination_slices = mappings[local_index]
            patch_probability = probability_patch[destination_slices]
            patch_weight = PILOT_WINDOW_WEIGHT[destination_slices]
            probability_sum[source_slices] += patch_probability * patch_weight
            weight_sum[source_slices] += patch_weight
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    if not np.isfinite(probability).all() or bool((weight_sum <= 0).any()):
        raise RuntimeError(f'{subject_id}: invalid sliding-window output.')
    return probability.astype(np.float32), target_binary, spacing


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap_pairs = np.column_stack([
        predicted_labels[target & prediction],
        target_labels[target & prediction],
    ])
    overlap_pairs = overlap_pairs[
        (overlap_pairs[:, 0] > 0) & (overlap_pairs[:, 1] > 0)
    ]
    if len(overlap_pairs) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap_pairs, axis=0, return_counts=True)
    order = np.argsort(-counts)
    used_predicted = set()
    used_target = set()
    matches = 0
    for pair_index in order:
        predicted_id, target_id = [int(value) for value in pairs[pair_index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


PILOT_THRESHOLDS = np.round(np.arange(0.10, 0.81, 0.05), 2)


@torch.no_grad()
def pilot_validate(study_model, epoch):
    case_cache = []
    for case_index, row in enumerate(heldout_rows, start=1):
        probability, target, spacing = pilot_full_volume_probability(
            study_model, row
        )
        case_cache.append({
            'row': row,
            'probability': probability.astype(np.float16),
            'target': target,
            'spacing': spacing,
        })
        print(
            f'  Held-out inference {case_index:02d}/{len(heldout_rows)}: '
            f'{row["subject_id"]}'
        )
    threshold_rows = []
    case_threshold_rows = []
    for threshold in PILOT_THRESHOLDS:
        positive_dice = []
        global_intersection = 0
        global_prediction = 0
        global_target = 0
        for case in case_cache:
            row = case['row']
            prediction = case['probability'] >= threshold
            target = case['target']
            intersection = int((prediction & target).sum())
            predicted_voxels = int(prediction.sum())
            target_voxels = int(target.sum())
            dice = (
                (2.0 * intersection) / (predicted_voxels + target_voxels)
                if predicted_voxels + target_voxels > 0
                else 1.0
            )
            hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
            if hard_metric_eligible and target_voxels > 0:
                positive_dice.append(dice)
                global_intersection += intersection
                global_prediction += predicted_voxels
                global_target += target_voxels
            case_threshold_rows.append({
                'epoch': epoch,
                'subject_id': str(row['subject_id']),
                'threshold': float(threshold),
                'hard_metric_eligible': hard_metric_eligible,
                'target_voxels': target_voxels,
                'predicted_voxels': predicted_voxels,
                'intersection_voxels': intersection,
                'dice': float(dice),
            })
        if not positive_dice:
            raise RuntimeError('Fold-0 held-out set has no positive hard targets.')
        global_dice = (
            2.0 * global_intersection
            / max(global_prediction + global_target, 1)
        )
        threshold_rows.append({
            'threshold': float(threshold),
            'positive_mean_dice': float(np.mean(positive_dice)),
            'positive_median_dice': float(np.median(positive_dice)),
            'positive_global_dice': float(global_dice),
            'positive_subjects': len(positive_dice),
        })
    threshold_df = pd.DataFrame(threshold_rows).sort_values(
        ['positive_mean_dice', 'positive_global_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected_threshold = float(threshold_df.iloc[0]['threshold'])
    selected_case_rows = []
    lesion_tp = lesion_fp = lesion_fn = 0
    total_intersection = total_prediction = total_target = 0
    positive_dice = []
    precisions = []
    recalls = []
    volume_errors = []
    empty_fp_volumes = []
    bin_dice = {name: [] for name in POSITIVE_BINS}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        prediction = probability >= selected_threshold
        target = case['target']
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0
            else 1.0
        )
        precision = intersection / max(predicted_voxels, 1)
        recall = intersection / max(target_voxels, 1)
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        predicted_volume_ml = predicted_voxels * voxel_volume_ml
        target_volume_ml = target_voxels * voxel_volume_ml
        hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
        true_positive_lesions = false_positive_lesions = false_negative_lesions = 0
        if hard_metric_eligible:
            (
                true_positive_lesions,
                false_positive_lesions,
                false_negative_lesions,
            ) = pilot_lesion_counts(prediction, target)
            lesion_tp += true_positive_lesions
            lesion_fp += false_positive_lesions
            lesion_fn += false_negative_lesions
            volume_errors.append(abs(predicted_volume_ml - target_volume_ml))
            if target_voxels > 0:
                positive_dice.append(dice)
                precisions.append(precision)
                recalls.append(recall)
                total_intersection += intersection
                total_prediction += predicted_voxels
                total_target += target_voxels
                lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
                if lesion_bin in bin_dice:
                    bin_dice[lesion_bin].append(dice)
            else:
                empty_fp_volumes.append(predicted_volume_ml)
        selected_case_rows.append({
            'epoch': epoch,
            'subject_id': str(row['subject_id']),
            'threshold': selected_threshold,
            'supervision_mode': str(row['trial8_supervision_mode']),
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': hard_metric_eligible,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'precision': float(precision),
            'recall': float(recall),
            'target_volume_ml': target_volume_ml,
            'predicted_volume_ml': predicted_volume_ml,
            'absolute_volume_error_ml': abs(
                predicted_volume_ml - target_volume_ml
            ),
            'lesion_true_positive': true_positive_lesions,
            'lesion_false_positive': false_positive_lesions,
            'lesion_false_negative': false_negative_lesions,
        })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    result = {
        'threshold': selected_threshold,
        'positive_mean_dice': float(np.mean(positive_dice)),
        'positive_median_dice': float(np.median(positive_dice)),
        'positive_global_dice': float(
            2.0 * total_intersection / max(total_prediction + total_target, 1)
        ),
        'mean_voxel_precision': float(np.mean(precisions)),
        'mean_voxel_recall': float(np.mean(recalls)),
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'mean_absolute_volume_error_ml': float(np.mean(volume_errors)),
        'mean_empty_false_positive_volume_ml': float(
            np.mean(empty_fp_volumes) if empty_fp_volumes else 0.0
        ),
        'positive_subjects': len(positive_dice),
        'hard_metric_subjects': int(sum(
            as_bool(case['row']['trial8_hard_metric_eligible'])
            for case in case_cache
        )),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(threshold_df, epoch_root / 'threshold_summary.csv')
    atomic_write_csv(
        pd.DataFrame(case_threshold_rows), epoch_root / 'case_threshold_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(selected_case_rows), epoch_root / 'selected_case_metrics.csv'
    )
    atomic_write_json(result, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    return result






MAX_EPOCHS = int(fivefold_policy['maximum_epochs'])
PATIENCE_VALIDATIONS = int(
    fivefold_policy['early_stopping_patience_validations']
)
EARLY_STOPPING_MINIMUM_EPOCH = int(
    fivefold_policy['early_stopping_minimum_epoch']
)
EARLY_STOPPING_MINIMUM_DELTA = float(
    fivefold_policy['early_stopping_minimum_delta']
)
BASE_LEARNING_RATE = float(fivefold_policy['learning_rate'])
WEIGHT_DECAY = float(fivefold_policy['weight_decay'])
ACCUMULATION_STEPS = int(fivefold_policy['gradient_accumulation_steps'])
GRADIENT_CLIP_NORM = float(fivefold_policy['gradient_clip_norm'])
EMA_DECAY = float(fivefold_policy['ema_decay'])
BATCH_SIZE = int(fivefold_policy['batch_size'])
GATE_LEARNING_RATE_MULTIPLIER = 5.0
NUM_WORKERS = 0

pilot_policy_checks = {
    'patch_size_matches': tuple(fivefold_policy['patch_size']) == PILOT_PATCH_SIZE,
    'batch_size_is_one': BATCH_SIZE == 1,
    'accumulation_steps_are_four': ACCUMULATION_STEPS == 4,
    'maximum_epochs_are_60': MAX_EPOCHS == 60,
    'validation_every_epoch': int(
        fivefold_policy['validation_interval_epochs']
    ) == 1,
    'metrics_printed_every_epoch': fivefold_policy[
        'metrics_printed_every_epoch'
    ] is True,
    'early_stopping_minimum_epoch_is_20': (
        EARLY_STOPPING_MINIMUM_EPOCH == 20
    ),
    'early_stopping_patience_is_15': PATIENCE_VALIDATIONS == 15,
    'early_stopping_delta_is_0p002': math.isclose(
        EARLY_STOPPING_MINIMUM_DELTA, 0.002
    ),
    'volume_case_weights_disabled': fivefold_policy[
        'volume_case_loss_weighting'
    ] is False,
    'top_k_ohem_disabled': fivefold_policy['loss']['top_k_ohem'] is False,
    'left_right_reflection_disabled': fivefold_policy[
        'left_right_reflection'
    ] is False,
    'checkpoint_metric_matches': fivefold_policy[
        'checkpoint_metric'
    ] == 'heldout_positive_mean_dice',
    'locked_validation_policy_preserved': 'do not read' in fivefold_policy[
        'locked_validation_policy'
    ],
    'test_policy_preserved': 'sealed' in fivefold_policy['test_policy'],
}
if not all(pilot_policy_checks.values()):
    failed = [name for name, passed in pilot_policy_checks.items() if not passed]
    raise RuntimeError('Frozen pilot policy changed: ' + ', '.join(failed))

set_determinism(PILOT_SEED)
training_dataset = Trial8Fold0PatchDataset()
epoch1_schedule = training_dataset.schedule
schedule_rows = [{
    'sample_index': index,
    'sample_type': entry['sample_type'],
    'subject_id': entry['subject_id'],
    'center_x': int(entry['center'][0]),
    'center_y': int(entry['center'][1]),
    'center_z': int(entry['center'][2]),
    'topology_relation': entry['topology_relation'],
} for index, entry in enumerate(epoch1_schedule)]
atomic_write_csv(pd.DataFrame(schedule_rows), PILOT_SCHEDULE_AUDIT_PATH)

pilot_schedule_checks = {
    'epoch_has_168_patches': len(epoch1_schedule) == 168,
    'positive_patches_are_112': training_dataset.counts['positive'] == 112,
    'topology_negative_patches_are_28': (
        training_dataset.counts['topology_negative'] == 28
    ),
    'brain_negative_patches_are_28': (
        training_dataset.counts['brain_negative'] == 28
    ),
    'positive_fraction_is_two_thirds': math.isclose(
        training_dataset.counts['positive'] / len(epoch1_schedule),
        2.0 / 3.0,
    ),
    'all_fit_positive_subjects_covered': set(
        fold0_fit_df.loc[
            fold0_fit_df['trial8_sampling_positive'].map(as_bool), 'subject_id'
        ].astype(str)
    ).issubset({
        entry['subject_id'] for entry in epoch1_schedule
        if entry['sample_type'] == 'positive'
    }),
}
if not all(pilot_schedule_checks.values()):
    failed = [name for name, passed in pilot_schedule_checks.items() if not passed]
    raise RuntimeError('Fold-0 epoch schedule changed: ' + ', '.join(failed))

training_model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=15,
    territory_feature_dim=65,
    vessel_feature_dim=15,
    case_feature_dim=8,
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)
if parameter_count(training_model) != EXPECTED_V2_PARAMETERS:
    raise RuntimeError('Fold-0 pilot model parameter count changed.')

smoke_loader = torch.utils.data.DataLoader(
    training_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == 'cuda',
)
smoke_cpu_batch = next(iter(smoke_loader))
training_model.train()
with torch.autocast(
    device_type=device.type,
    dtype=torch.bfloat16,
    enabled=device.type == 'cuda',
):
    pilot_smoke_loss, pilot_smoke_parts = pilot_compute_training_loss(
        training_model, smoke_cpu_batch
    )
if not bool(torch.isfinite(pilot_smoke_loss).item()):
    raise RuntimeError('Fold-0 pilot smoke loss is non-finite.')
pilot_smoke_loss.backward()
pilot_smoke_gradient_finite = all(
    parameter.grad is None or bool(torch.isfinite(parameter.grad).all().item())
    for parameter in training_model.parameters()
)
pilot_smoke_gradient_nonzero = any(
    parameter.grad is not None and bool((parameter.grad != 0).any().item())
    for parameter in training_model.parameters()
)
training_model.zero_grad(set_to_none=True)

pilot_start_checks = {
    **pilot_upstream_checks,
    **pilot_manifest_checks,
    **pilot_scaler_contract_checks,
    **pilot_target_checks,
    **pilot_policy_checks,
    **pilot_schedule_checks,
    'model_parameter_count_matches': parameter_count(training_model)
    == EXPECTED_V2_PARAMETERS,
    'smoke_loss_finite': bool(torch.isfinite(pilot_smoke_loss).item()),
    'smoke_gradients_finite': pilot_smoke_gradient_finite,
    'smoke_has_nonzero_gradient': pilot_smoke_gradient_nonzero,
    'target_access_is_development_train_only': True,
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
    'outcome_workbooks_not_read': True,
    'expert_routing_disabled': True,
    'direct_graph_to_logit_residual_disabled': True,
}
pilot_training_authorized = bool(all(pilot_start_checks.values()))
atomic_write_json({
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'pilot_version': PILOT_VERSION,
    'code_revision': PILOT_CODE_REVISION,
    'pilot_training_authorized': pilot_training_authorized,
    'checks': pilot_start_checks,
    'fold': HELDOUT_FOLD,
    'fit_subjects': len(fold0_fit_df),
    'heldout_subjects': len(fold0_heldout_df),
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'outcome_workbooks_read': False,
    'full_fivefold_training_authorized': False,
}, PILOT_START_GATE_PATH)
if not pilot_training_authorized:
    failed = [name for name, passed in pilot_start_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 FOLD-0 PILOT START GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ema_model = copy.deepcopy(training_model).to(device).eval()
ema_model.requires_grad_(False)
gate_parameters = []
standard_parameters = []
for name, parameter in training_model.named_parameters():
    if '.gate.network.2.' in name:
        gate_parameters.append(parameter)
    else:
        standard_parameters.append(parameter)
if not gate_parameters:
    raise RuntimeError('Zero-initialized fusion-gate parameters were not found.')
training_optimizer = torch.optim.AdamW([
    {
        'params': standard_parameters,
        'lr': BASE_LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'group_name': 'standard',
    },
    {
        'params': gate_parameters,
        'lr': BASE_LEARNING_RATE * GATE_LEARNING_RATE_MULTIPLIER,
        'weight_decay': 0.0,
        'group_name': 'zero_init_fusion_gates',
    },
])
updates_per_epoch = math.ceil(len(training_dataset) / ACCUMULATION_STEPS)
total_updates = MAX_EPOCHS * updates_per_epoch
warmup_updates = 3 * updates_per_epoch


def pilot_lr_lambda(update_index):
    completed = update_index + 1
    if completed <= warmup_updates:
        return completed / max(warmup_updates, 1)
    progress = (completed - warmup_updates) / max(
        total_updates - warmup_updates, 1
    )
    return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


training_scheduler = torch.optim.lr_scheduler.LambdaLR(
    training_optimizer, lr_lambda=pilot_lr_lambda
)


def pilot_update_ema(ema, study, maximum_decay, update_step):
    decay = min(maximum_decay, (1.0 + update_step) / (10.0 + update_step))
    with torch.no_grad():
        for ema_parameter, parameter in zip(
            ema.parameters(), study.parameters()
        ):
            ema_parameter.mul_(decay).add_(parameter, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(ema.buffers(), study.buffers()):
            ema_buffer.copy_(buffer)


def pilot_epoch_loader(epoch):
    training_dataset.set_epoch(epoch)
    generator = torch.Generator()
    generator.manual_seed(PILOT_SEED + epoch * 97_409)
    return torch.utils.data.DataLoader(
        training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda',
        persistent_workers=False,
        generator=generator,
    )


pilot_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'pilot_version': PILOT_VERSION,
    'code_revision': PILOT_CODE_REVISION,
    'fold': HELDOUT_FOLD,
    'fit_subject_count': len(fold0_fit_df),
    'heldout_subject_count': len(fold0_heldout_df),
    'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
    'heldout_subject_hash': pilot_subject_hash(fold0_heldout_df['subject_id']),
    'multimodal_manifest_sha256': multimodal_manifest_sha256,
    'fold_manifest_sha256': fold_manifest_sha256,
    'loader_gate_sha256': loader_gate_sha256,
    'fivefold_gate_sha256': fivefold_gate_sha256,
    'v2_model_gate_sha256': v2_model_gate_sha256,
    'architecture_sha256': architecture_sha256,
    'architecture_contract_sha256': architecture_contract_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'fold0_graph_scaler_sha256': fold0_graph_scaler_sha256,
    'fold0_clinical_scaler_sha256': fold0_clinical_scaler_sha256,
    'parameters': EXPECTED_V2_PARAMETERS,
    'patch_size': list(PILOT_PATCH_SIZE),
    'samples_per_epoch': len(training_dataset),
    'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': ACCUMULATION_STEPS,
    'maximum_epochs': MAX_EPOCHS,
    'base_learning_rate': BASE_LEARNING_RATE,
    'gate_learning_rate_multiplier': GATE_LEARNING_RATE_MULTIPLIER,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_decay': EMA_DECAY,
    'early_stopping_minimum_epoch': EARLY_STOPPING_MINIMUM_EPOCH,
    'early_stopping_patience': PATIENCE_VALIDATIONS,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'checkpoint_metric': 'heldout_positive_mean_dice',
    'threshold_grid': PILOT_THRESHOLDS.tolist(),
    'mixed_precision': 'bfloat16_on_cuda_float32_reductions',
    'spatial_augmentation_maximum_attempts': 4,
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'full_fivefold_training_authorized': False,
}
if PILOT_CONFIG_PATH.exists():
    existing_config = json.loads(PILOT_CONFIG_PATH.read_text(encoding='utf-8'))
    immutable_keys = [
        'pilot_version', 'code_revision', 'fold',
        'fit_subject_hash', 'heldout_subject_hash',
        'multimodal_manifest_sha256', 'fold_manifest_sha256',
        'loader_gate_sha256', 'fivefold_gate_sha256',
        'architecture_contract_sha256', 'graph_schema_sha256',
        'fold0_graph_scaler_sha256', 'fold0_clinical_scaler_sha256',
        'parameters', 'patch_size', 'samples_per_epoch', 'batch_size',
        'gradient_accumulation_steps', 'maximum_epochs',
        'base_learning_rate', 'gate_learning_rate_multiplier',
        'weight_decay', 'gradient_clip_norm', 'ema_decay',
        'early_stopping_minimum_epoch', 'early_stopping_patience',
        'early_stopping_minimum_delta', 'checkpoint_metric',
        'threshold_grid', 'mixed_precision',
        'spatial_augmentation_maximum_attempts',
    ]
    if any(
        existing_config.get(key) != pilot_config.get(key)
        for key in immutable_keys
    ):
        raise RuntimeError('Existing fold-0 pilot configuration changed.')
else:
    atomic_write_json(pilot_config, PILOT_CONFIG_PATH)

start_epoch = 1
global_update_step = 0
best_mean_dice = -math.inf
best_epoch = 0
best_threshold = 0.50
validations_without_improvement = 0
history_rows = []
if PILOT_LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    checkpoint_checks = {
        'pilot_version_matches': checkpoint.get('pilot_version') == PILOT_VERSION,
        'code_revision_matches': checkpoint.get('code_revision')
        == PILOT_CODE_REVISION,
        'manifest_hash_matches': checkpoint.get('multimodal_manifest_sha256')
        == multimodal_manifest_sha256,
        'loader_gate_hash_matches': checkpoint.get('loader_gate_sha256')
        == loader_gate_sha256,
        'graph_scaler_hash_matches': checkpoint.get(
            'fold0_graph_scaler_sha256'
        ) == fold0_graph_scaler_sha256,
        'clinical_scaler_hash_matches': checkpoint.get(
            'fold0_clinical_scaler_sha256'
        ) == fold0_clinical_scaler_sha256,
        'architecture_contract_hash_matches': checkpoint.get(
            'architecture_contract_sha256'
        ) == architecture_contract_sha256,
        'parameter_count_matches': checkpoint.get('parameters')
        == EXPECTED_V2_PARAMETERS,
        'locked_validation_targets_not_read': checkpoint.get(
            'locked_validation_targets_read'
        ) is False,
        'test_targets_not_read': checkpoint.get('test_targets_read') is False,
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Fold-0 checkpoint mismatch: ' + ', '.join(failed))
    training_model.load_state_dict(checkpoint['model_state_dict'])
    ema_model.load_state_dict(checkpoint['ema_model_state_dict'])
    training_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    training_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    pilot_restore_rng_state(checkpoint['rng_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    global_update_step = int(checkpoint['global_update_step'])
    best_mean_dice = float(checkpoint['best_mean_dice'])
    best_epoch = int(checkpoint['best_epoch'])
    best_threshold = float(checkpoint['best_threshold'])
    validations_without_improvement = int(
        checkpoint['validations_without_improvement']
    )
    history_rows = list(checkpoint.get('history_rows', []))
    print(
        f'Resuming fold-0 pilot after epoch {start_epoch - 1}; '
        f'best EMA Dice={best_mean_dice:.4f} at epoch {best_epoch}.'
    )
else:
    print('Starting fold-0 pilot from a fresh deterministic initialization.')






print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 FOLD-0 PILOT TRAINING START')
print('=' * 80)
print(f'Device / GPU:                       {device} / ' + (
    torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'
))
print(f'Fit / held-out / locked val / test: {len(fold0_fit_df)} / {len(fold0_heldout_df)} / 0 / 0 targets')
print(f'Patch size:                         {PILOT_PATCH_SIZE}')
print(f'Samples per epoch:                  {len(training_dataset)} (112/28/28)')
print(f'Batch / accumulation / effective:   {BATCH_SIZE} / {ACCUMULATION_STEPS} / {BATCH_SIZE * ACCUMULATION_STEPS}')
print(f'Maximum epochs:                     {MAX_EPOCHS}')
print(f'Early stopping:                     epoch >= {EARLY_STOPPING_MINIMUM_EPOCH}, patience={PATIENCE_VALIDATIONS}, delta={EARLY_STOPPING_MINIMUM_DELTA:.3f}')
print(f'Base/gate learning rates:           {BASE_LEARNING_RATE:.1e} / {BASE_LEARNING_RATE * GATE_LEARNING_RATE_MULTIPLIER:.1e}')
print(f'Total parameters:                   {EXPECTED_V2_PARAMETERS:,}')
print('Validation:                         full-volume EMA every epoch')
print('Threshold calibration:              held-out fold only, 0.10..0.80')
print('Locked validation/test targets:     sealed')

training_already_complete = (
    start_epoch > MAX_EPOCHS
    or (
        start_epoch - 1 >= EARLY_STOPPING_MINIMUM_EPOCH
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    )
)
stop_reason = (
    'already_complete_on_resume'
    if training_already_complete
    else 'maximum_epochs_reached'
)

for epoch in range(
    start_epoch,
    start_epoch if training_already_complete else MAX_EPOCHS + 1,
):
    epoch_start = datetime.now(timezone.utc)
    loader = pilot_epoch_loader(epoch)
    expected_counts = dict(training_dataset.counts)
    training_model.train()
    training_optimizer.zero_grad(set_to_none=True)
    running_names = [
        'total_loss', 'main_loss', 'soft_dice_loss',
        'soft_focal_bce_loss', 'hard_focal_tversky_loss',
        'hard_contribution', 'half_loss', 'quarter_loss', 'node_loss',
        'atlas_retained_fraction', 'vascular_retained_fraction',
        'occlusion_retained_fraction', 'clinical_retained_fraction',
        'spatially_augmented_fraction',
    ]
    running = {name: 0.0 for name in running_names}
    processed_batches = 0
    processed_samples = 0
    observed_counts = {
        'positive': 0, 'topology_negative': 0, 'brain_negative': 0,
    }
    gradient_norms = []
    accumulation_subjects = []
    for batch_index, cpu_batch in enumerate(loader, start=1):
        processed_samples += int(cpu_batch['target_binary'].shape[0])
        for sample_type in cpu_batch['sample_type']:
            observed_counts[str(sample_type)] += 1
        accumulation_subjects.extend(str(value) for value in cpu_batch['subject_id'])
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            total_loss, parts = pilot_compute_training_loss(
                training_model, cpu_batch
            )
            group_start = ((batch_index - 1) // ACCUMULATION_STEPS) * (
                ACCUMULATION_STEPS
            )
            group_size = min(
                ACCUMULATION_STEPS, len(loader) - group_start
            )
            scaled_loss = total_loss / group_size
        scaled_loss.backward()
        for name in running_names:
            running[name] += float(parts[name].detach().cpu())
        processed_batches += 1
        should_update = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(loader)
        )
        if should_update:
            gradient_norm = float(torch.nn.utils.clip_grad_norm_(
                training_model.parameters(), GRADIENT_CLIP_NORM
            ).item())
            if not math.isfinite(gradient_norm):
                training_optimizer.zero_grad(set_to_none=True)
                raise RuntimeError(
                    f'Non-finite gradient at epoch {epoch}, batch '
                    f'{batch_index}, subjects={accumulation_subjects}.'
                )
            gradient_norms.append(gradient_norm)
            training_optimizer.step()
            training_optimizer.zero_grad(set_to_none=True)
            training_scheduler.step()
            global_update_step += 1
            pilot_update_ema(
                ema_model, training_model, EMA_DECAY, global_update_step
            )
            accumulation_subjects.clear()
    if observed_counts != expected_counts or processed_samples != len(training_dataset):
        raise RuntimeError(
            f'Epoch {epoch}: sampling counts changed. '
            f'Observed {observed_counts}; expected {expected_counts}.'
        )
    epoch_means = {
        name: running[name] / max(processed_batches, 1)
        for name in running_names
    }
    if not gradient_norms or not all(
        math.isfinite(value) for value in epoch_means.values()
    ):
        raise RuntimeError(f'Epoch {epoch}: non-finite training state.')

    print(f'Running full fold-0 EMA validation at epoch {epoch}...')
    validation_result = pilot_validate(ema_model, epoch)
    current_dice = float(validation_result['positive_mean_dice'])
    current_threshold = float(validation_result['threshold'])
    improved = current_dice > best_mean_dice + EARLY_STOPPING_MINIMUM_DELTA
    if improved:
        best_mean_dice = current_dice
        best_epoch = epoch
        best_threshold = current_threshold
        validations_without_improvement = 0
    else:
        validations_without_improvement += 1
    epoch_seconds = (
        datetime.now(timezone.utc) - epoch_start
    ).total_seconds()
    history_row = {
        'epoch': epoch,
        'global_update_step': global_update_step,
        'standard_learning_rate': float(training_optimizer.param_groups[0]['lr']),
        'fusion_gate_learning_rate': float(training_optimizer.param_groups[1]['lr']),
        **epoch_means,
        'mean_gradient_norm': float(np.mean(gradient_norms)),
        'maximum_gradient_norm': float(np.max(gradient_norms)),
        'positive_patch_fraction': observed_counts['positive'] / processed_samples,
        'topology_negative_fraction': (
            observed_counts['topology_negative'] / processed_samples
        ),
        'brain_negative_fraction': (
            observed_counts['brain_negative'] / processed_samples
        ),
        'validation_threshold': current_threshold,
        'validation_positive_mean_dice': current_dice,
        'validation_positive_median_dice': validation_result[
            'positive_median_dice'
        ],
        'validation_positive_global_dice': validation_result[
            'positive_global_dice'
        ],
        'validation_mean_voxel_precision': validation_result[
            'mean_voxel_precision'
        ],
        'validation_mean_voxel_recall': validation_result[
            'mean_voxel_recall'
        ],
        'validation_lesion_precision': validation_result['lesion_precision'],
        'validation_lesion_recall': validation_result['lesion_recall'],
        'validation_lesion_f1': validation_result['lesion_f1'],
        'validation_mean_absolute_volume_error_ml': validation_result[
            'mean_absolute_volume_error_ml'
        ],
        'validation_mean_empty_false_positive_volume_ml': validation_result[
            'mean_empty_false_positive_volume_ml'
        ],
        'best_positive_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'improved': improved,
        'epoch_seconds': epoch_seconds,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    for lesion_bin, value in validation_result['lesion_bin_mean_dice'].items():
        history_row[f'validation_mean_dice_{lesion_bin}'] = value
    history_rows.append(to_builtin(history_row))
    atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)

    checkpoint_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'epoch': epoch,
        'global_update_step': global_update_step,
        'model_state_dict': training_model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': training_optimizer.state_dict(),
        'scheduler_state_dict': training_scheduler.state_dict(),
        'rng_state': pilot_capture_rng_state(),
        'best_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'history_rows': history_rows,
        'multimodal_manifest_sha256': multimodal_manifest_sha256,
        'loader_gate_sha256': loader_gate_sha256,
        'fivefold_gate_sha256': fivefold_gate_sha256,
        'architecture_sha256': architecture_sha256,
        'architecture_contract_sha256': architecture_contract_sha256,
        'fold0_graph_scaler_sha256': fold0_graph_scaler_sha256,
        'fold0_clinical_scaler_sha256': fold0_clinical_scaler_sha256,
        'parameters': EXPECTED_V2_PARAMETERS,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
    }
    pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
    if improved:
        pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)

    print(
        f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
        f'loss={epoch_means["total_loss"]:.4f}, '
        f'main={epoch_means["main_loss"]:.4f}, '
        f'node={epoch_means["node_loss"]:.4f}, '
        f'lr={history_row["standard_learning_rate"]:.2e}, '
        f'gate-lr={history_row["fusion_gate_learning_rate"]:.2e}, '
        f'grad={history_row["mean_gradient_norm"]:.3f}, '
        f'time={epoch_seconds / 60.0:.1f} min'
    )
    print(
        f'  EMA held-out: mean Dice={current_dice:.4f}, '
        f'median Dice={validation_result["positive_median_dice"]:.4f}, '
        f'global Dice={validation_result["positive_global_dice"]:.4f}, '
        f'voxel P/R={validation_result["mean_voxel_precision"]:.4f}/'
        f'{validation_result["mean_voxel_recall"]:.4f}, '
        f'lesion F1={validation_result["lesion_f1"]:.4f}, '
        f'threshold={current_threshold:.2f}'
    )
    print(
        f'  Volume MAE={validation_result["mean_absolute_volume_error_ml"]:.3f} mL, '
        f'empty FP={validation_result["mean_empty_false_positive_volume_ml"]:.3f} mL, '
        f'best={best_mean_dice:.4f} at epoch {best_epoch}, '
        f'patience={validations_without_improvement}/{PATIENCE_VALIDATIONS}'
    )
    printable_bins = [
        f'{name}={value:.4f}'
        for name, value in validation_result['lesion_bin_mean_dice'].items()
        if math.isfinite(value)
    ]
    print('  Mean Dice by lesion bin: ' + ', '.join(printable_bins))

    atomic_write_json({
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'status': 'running',
        'last_completed_epoch': epoch,
        'best_positive_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
    }, PILOT_STATE_PATH)
    if (
        epoch >= EARLY_STOPPING_MINIMUM_EPOCH
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    ):
        stop_reason = 'early_stopping'
        print(
            f'Early stopping after {validations_without_improvement} '
            'held-out validations without sufficient improvement.'
        )
        break

completed_epoch = int(history_rows[-1]['epoch']) if history_rows else start_epoch - 1
pilot_complete = bool(
    completed_epoch >= MAX_EPOCHS or stop_reason in {
        'early_stopping', 'already_complete_on_resume'
    }
)
result_gate = {
    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    'pilot_version': PILOT_VERSION,
    'code_revision': PILOT_CODE_REVISION,
    'pilot_complete': pilot_complete,
    'stop_reason': stop_reason,
    'last_completed_epoch': completed_epoch,
    'best_positive_mean_dice': best_mean_dice,
    'best_epoch': best_epoch,
    'best_threshold': best_threshold,
    'fit_subjects': len(fold0_fit_df),
    'heldout_subjects': len(fold0_heldout_df),
    'parameters': EXPECTED_V2_PARAMETERS,
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'outcome_workbooks_read': False,
    'pilot_review_required': True,
    'full_fivefold_training_authorized': False,
    'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
    'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
    'history_path': str(PILOT_HISTORY_PATH),
}
atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
atomic_write_json({
    **result_gate,
    'status': 'complete' if pilot_complete else 'interrupted',
}, PILOT_STATE_PATH)

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 FOLD-0 PILOT COMPLETE')
print('=' * 80)
print(f'Stop reason:                        {stop_reason}')
print(f'Last completed epoch:               {completed_epoch}')
print(f'Best EMA positive mean Dice:        {best_mean_dice:.4f}')
print(f'Best epoch:                         {best_epoch}')
print(f'Calibrated threshold:               {best_threshold:.2f}')
print('Validation cadence:                 every epoch')
print('Locked validation targets read:     False')
print('Test targets read:                  False')
print('Full five-fold training authorized: False (pilot review required)')
print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
print(f'Last checkpoint:                    {PILOT_LAST_CHECKPOINT_PATH}')
print(f'History:                            {PILOT_HISTORY_PATH}')
print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


In [ ]:
import sys
import os
import gc
import copy
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')
scipy = ensure_package('scipy')

import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AVAILABILITY_VERSION = 'v1_feature_availability_and_test_time_contract'
PREFLIGHT_VERSION = 'v2_1_isolated_zero_init_gated_fusion_construction'
CODE_REVISION = 'r3_pytorch_module_namespace_safe'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_BASELINE_PARAMETERS = 15_052_628

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

AVAILABILITY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / AVAILABILITY_VERSION
)
AVAILABILITY_GATE_PATH = AVAILABILITY_ROOT / 'feature_availability_gate.json'
INPUT_CONTRACT_PATH = AVAILABILITY_ROOT / 'architecture_input_contract.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
LEGACY_V2_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / 'v1_zero_init_gated_fusion_construction'
)
if OUTPUT_ROOT.resolve() == LEGACY_V2_PREFLIGHT_ROOT.resolve():
    raise RuntimeError(
        'V2.1 construction output overlaps the legacy V2 preflight root.'
    )
SCALER_NPZ_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = OUTPUT_ROOT / 'architecture_config.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_model_construction_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_model_construction_audit.csv'

SEED = 20260927
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_428_468
PARAMETER_BUDGET_MINIMUM = 15_300_000
PARAMETER_BUDGET_MAXIMUM = 15_650_000

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return to_builtin(value.tolist())
    if isinstance(value, np.generic):
        return to_builtin(value.item())
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    AVAILABILITY_GATE_PATH,
    INPUT_CONTRACT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
availability_gate = json.loads(
    AVAILABILITY_GATE_PATH.read_text(encoding='utf-8')
)
input_contract = json.loads(INPUT_CONTRACT_PATH.read_text(encoding='utf-8'))
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')

if availability_gate.get('feature_availability_audit_passed') is not True:
    raise RuntimeError('Vascular-fusion feature-availability gate is not passed.')
if availability_gate.get('architecture_implementation_authorized') is not True:
    raise RuntimeError('Feature audit does not authorize architecture construction.')
if availability_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Feature audit must leave model training locked.')
if availability_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Feature audit did not remain target-free.')
if availability_gate.get('outcome_workbooks_read') is not False:
    raise RuntimeError('Feature audit read outcome workbooks.')
if availability_gate.get('manifest_sha256') != EXPECTED_TARGET_POLICY_MANIFEST_SHA256:
    raise RuntimeError('Feature audit target-policy manifest hash mismatch.')
if availability_gate.get('clinical_token_input_dimension') != CLINICAL_INPUT_DIM:
    raise RuntimeError('Clinical-token input dimension changed.')
if input_contract.get('architecture_version') != MODEL_VERSION:
    raise RuntimeError('Architecture input-contract version mismatch.')
if input_contract.get('model_training_authorized') is not False:
    raise RuntimeError('Input contract must leave model training locked.')
if input_contract.get('anti_leakage', {}).get('target_arrays_read') is not False:
    raise RuntimeError('Input contract did not remain target-free.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print(f'Trial8 v2 model preflight revision: {CODE_REVISION}')
print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Vascular-fusion availability gate: passed')
print('Target arrays read: False')
print('Outcome workbooks read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

atomic_write_npz(
    SCALER_NPZ_PATH,
    territory_feature_names=np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    territory_mean=territory_mean,
    territory_std=territory_std,
    territory_count=territory_count,
    vessel_feature_names=np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    vessel_source_indices=np.asarray(vessel_source_indices, dtype=np.int16),
    vessel_mean=vessel_mean,
    vessel_std=vessel_std,
    vessel_count=vessel_count,
    case_feature_names=np.asarray(MODEL_CASE_FEATURE_NAMES),
    case_source_indices=np.asarray(case_source_indices, dtype=np.int16),
    case_scale_mask=case_scale_mask.astype(np.uint8),
    case_mean=case_mean,
    case_std=case_std,
    case_count=case_count,
    clip_value=np.asarray(SCALER_CLIP, dtype=np.float32),
    fit_split=np.asarray('train'),
    fit_subject_count=np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    fit_subject_hash=np.asarray(scaler_fit_subject_hash),
    target_arrays_read=np.asarray(False, dtype=np.uint8),
)
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(scaler_record, SCALER_JSON_PATH)

print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    

    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(
                math.atanh(float(fraction_of_bound))
            )


class AsymmetricVascularStream3D(nn.Module):
    

    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        pooled = v5.mean(dim=(2, 3, 4))
        return v3, v4, v5, pooled


class SpatialConfidenceFusion3D(nn.Module):
    

    def __init__(
        self,
        vascular_channels,
        image_channels,
        confidence_dim=4,
        bound=0.1,
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, kernel_size=1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound
        )

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature,
                size=image_feature.shape[2:],
                mode='trilinear',
                align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.value_dim = int(value_dim)
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        safe_values = values * valid
        return self.encoder(torch.cat([safe_values, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    

    def __init__(
        self,
        vascular_dim,
        occlusion_dim,
        clinical_dim,
        output_dim=64,
        confidence_dim=4,
        bound=0.1,
        dropout=0.10,
    ):
        super().__init__()
        source_dim = vascular_dim + occlusion_dim + clinical_dim
        self.source = nn.Sequential(
            nn.Linear(source_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self,
        graph_context,
        vascular_context,
        occlusion_context,
        clinical_context,
        confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context,
            occlusion_context,
            clinical_context,
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
        vascular_channels=(8, 16, 32, 48),
        clinical_input_dim=44,
        clinical_embed_dim=32,
        occlusion_value_dim=7,
        occlusion_embed_dim=32,
        confidence_dim=4,
        fusion_bound=0.1,
        vascular_dropout=0.15,
        clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)

        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        vc1, vc2, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vascular_dim=vc4,
            occlusion_dim=occlusion_embed_dim,
            clinical_dim=clinical_embed_dim,
            output_dim=graph_hidden,
            confidence_dim=confidence_dim,
            bound=fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        modules = [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]
        for module in modules:
            module.activate_for_gradient_test(0.5)

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        vascular_image,
        confidence_features,
        occlusion_values,
        occlusion_valid,
        clinical_input,
        vascular_available,
        occlusion_available,
        clinical_available,
        force_atlas_dropout=None,
        force_vascular_dropout=None,
        force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available,
            self.vascular_dropout,
            force_vascular_dropout,
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available,
            self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available,
            self.clinical_dropout,
            force_clinical_dropout,
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        vascular_gate = effective_vascular.to(image.dtype)
        safe_vascular_image = (
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        v3, v4, v5, vascular_context = self.vascular_stream(
            safe_vascular_image
        )
        vascular_context = (
            vascular_context * vascular_gate[:, None]
        )
        safe_occlusion_valid = (
            occlusion_valid
            * effective_occlusion[:, None].to(occlusion_valid.dtype)
        )
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input)
        clinical_context = (
            clinical_context * effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context,
            vascular_context,
            occlusion_context,
            clinical_context,
            effective_confidence,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')


preflight_row = None
for _, candidate_row in preflight_candidates.iterrows():
    candidate_npz_path = Path(str(candidate_row['vascufate_v2_npz_path']))
    with np.load(candidate_npz_path, allow_pickle=False) as candidate_data:
        candidate_lvo_present = (
            'lvo_present' in candidate_data.files
            and bool(np.asarray(candidate_data['lvo_present']).reshape(-1)[0])
        )
    if candidate_lvo_present:
        preflight_row = candidate_row
        break
if preflight_row is None:
    raise RuntimeError(
        'No atlas-enabled train-only case has a valid LVO source for the '
        'complete v2 branch smoke test.'
    )
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    required_npz_inputs = {
        'image',
        'subject_id',
        'input_channels',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
    }
    missing_npz_inputs = sorted(required_npz_inputs - set(data.files))
    if missing_npz_inputs:
        raise RuntimeError(
            'Preflight source NPZ lacks required input arrays: '
            + ', '.join(missing_npz_inputs)
        )
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
    input_channels = decode_strings(data['input_channels'])
    cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
    lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
    lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')
if input_channels != EXPECTED_INPUT_CHANNELS:
    raise RuntimeError('Preflight input-channel schema changed.')
if cow_soft_union.shape != image.shape[1:]:
    raise RuntimeError('CoW prior shape differs from the image grid.')
if lvo_proximity.shape != image.shape[1:]:
    raise RuntimeError('LVO proximity shape differs from the image grid.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

cow_prior_valid = bool(
    case_features_source[
        SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')
    ] > 0.5
)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))
cow_patch = center_crop_or_pad(
    cow_soft_union, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
lvo_patch = center_crop_or_pad(
    lvo_proximity, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
vascular_patch = np.stack([
    image_patch[input_channels.index('cta')],
    image_patch[input_channels.index('tmax')],
    image_patch[input_channels.index('perfusion_valid')],
    cow_patch,
    lvo_patch,
], axis=0).astype(np.float32)




clinical_input = np.zeros(CLINICAL_INPUT_DIM, dtype=np.float32)
clinical_input[:22] = np.linspace(-0.5, 0.5, 22, dtype=np.float32)
clinical_input[22:] = (np.arange(22) % 3 == 0).astype(np.float32)

occlusion_feature_names = [
    'cta_mean',
    'cta_p90',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
occlusion_values = [float(lvo_present), float(lvo_patch.max())]
occlusion_valid = [1.0, float(lvo_present)]
for feature_name in occlusion_feature_names:
    feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
    valid_nodes = selected_vessel_valid[:, feature_index]
    if bool(valid_nodes.any()):
        occlusion_values.append(float(scaled_vessel[valid_nodes, feature_index].mean()))
        occlusion_valid.append(1.0)
    else:
        occlusion_values.append(0.0)
        occlusion_valid.append(0.0)
occlusion_values = np.asarray(occlusion_values, dtype=np.float32)
occlusion_valid = np.asarray(occlusion_valid, dtype=np.float32)
if occlusion_values.shape != (OCCLUSION_VALUE_DIM,):
    raise RuntimeError('Occlusion-source vector dimension changed.')

confidence_features = np.asarray([
    float(vascular_patch[2].mean()),
    float(cow_prior_valid),
    float(np.clip(selected_case[1], 0.0, 1.0)),
    float(lvo_present),
], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
    'vascular_image': torch.from_numpy(vascular_patch[None]).to(device),
    'confidence_features': torch.from_numpy(
        confidence_features[None]
    ).to(device),
    'occlusion_values': torch.from_numpy(occlusion_values[None]).to(device),
    'occlusion_valid': torch.from_numpy(occlusion_valid[None]).to(device),
    'clinical_input': torch.from_numpy(clinical_input[None]).to(device),
    'vascular_available': torch.ones(1, dtype=torch.bool, device=device),
    'occlusion_available': torch.as_tensor(
        [lvo_present], dtype=torch.bool, device=device
    ),
    'clinical_available': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
vascular_stream_parameters = parameter_count(model.vascular_stream)
spatial_fusion_parameters = parameter_count(
    model.vascular_fusion_quarter
) + parameter_count(model.vascular_fusion_eighth) + parameter_count(
    model.vascular_fusion_bottleneck
)
clinical_parameters = parameter_count(model.clinical_encoder)
occlusion_parameters = parameter_count(model.occlusion_encoder)
context_fusion_parameters = parameter_count(model.context_fusion)
new_fusion_parameters = (
    vascular_stream_parameters
    + spatial_fusion_parameters
    + clinical_parameters
    + occlusion_parameters
    + context_fusion_parameters
)
backbone_parameters = total_parameters - (
    graph_parameters
    + film_parameters
    + head_parameters
    + new_fusion_parameters
)
preserved_v1_parameters = (
    backbone_parameters + graph_parameters + film_parameters + head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'asymmetric vascular stream', 'parameters': vascular_stream_parameters},
    {'component': 'confidence-gated spatial fusion', 'parameters': spatial_fusion_parameters},
    {'component': 'clinical token encoder', 'parameters': clinical_parameters},
    {'component': 'occlusion source encoder', 'parameters': occlusion_parameters},
    {'component': 'gated context fusion', 'parameters': context_fusion_parameters},
    {'component': 'preserved v1 path', 'parameters': preserved_v1_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad():
    normal_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    atlas_dropped_output = model(
        **batch,
        force_atlas_dropout=True,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    new_sources_dropped_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=True,
        force_clinical_dropout=True,
    )

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
    'fused_global_context': (1, GRAPH_HIDDEN),
    'new_source_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(new_sources_dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        atlas_dropped_output['atlas_dropped'].item()
        and not atlas_dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'],
        atlas_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'initial_segmentation_is_new_sources_neutral': torch.allclose(
        normal_output['logits'],
        new_sources_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
    'zero_init_new_fusion_gates_are_zero': all(
        torch.count_nonzero(module.network[-1].weight).item() == 0
        and torch.count_nonzero(module.network[-1].bias).item() == 0
        for module in [
            model.vascular_fusion_quarter.gate,
            model.vascular_fusion_eighth.gate,
            model.vascular_fusion_bottleneck.gate,
            model.context_fusion.gate,
        ]
    ),
    'normal_vascular_source_enabled': bool(
        normal_output['effective_vascular_enabled'].item()
    ),
    'normal_clinical_source_enabled': bool(
        normal_output['effective_clinical_enabled'].item()
    ),
    'forced_new_source_dropout_applied': bool(
        new_sources_dropped_output['vascular_dropped'].item()
        and new_sources_dropped_output['occlusion_dropped'].item()
        and new_sources_dropped_output['clinical_dropped'].item()
    ),
})




model.activate_new_fusion_gates_for_gradient_test()
with torch.no_grad():
    nn.init.normal_(model.global_film.project[-1].weight, mean=0.0, std=1e-3)
    nn.init.zeros_(model.global_film.project[-1].bias)

model.eval()
with torch.no_grad():
    activated_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    vascular_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    occlusion_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=True,
        force_clinical_dropout=False,
    )
    clinical_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=True,
    )

ablation_differences = {
    'vascular_max_abs_logit_change': float(
        (activated_output['logits'] - vascular_ablation_output['logits'])
        .abs().max().item()
    ),
    'occlusion_max_abs_logit_change': float(
        (activated_output['logits'] - occlusion_ablation_output['logits'])
        .abs().max().item()
    ),
    'clinical_max_abs_logit_change': float(
        (activated_output['logits'] - clinical_ablation_output['logits'])
        .abs().max().item()
    ),
}
ablation_checks = {
    'activated_outputs_finite': all(
        torch.isfinite(activated_output[name]).all().item()
        for name in expected_shapes
    ),
    'vascular_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['vascular_max_abs_logit_change'] > 1e-8
    ),
    'occlusion_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['occlusion_max_abs_logit_change'] > 1e-8
    ),
    'clinical_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['clinical_max_abs_logit_change'] > 1e-8
    ),
    'all_activated_gates_remain_bounded': all(
        float(activated_output[name].abs().max().item()) <= FUSION_BOUND + 1e-6
        for name in [
            'context_fusion_gate',
            'vascular_quarter_gate',
            'vascular_eighth_gate',
            'vascular_bottleneck_gate',
        ]
    ),
}

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=False,
):
    gradient_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('vascular_stream', model.vascular_stream),
    ('spatial_fusion', nn.ModuleList([
        model.vascular_fusion_quarter,
        model.vascular_fusion_eighth,
        model.vascular_fusion_bottleneck,
    ])),
    ('clinical_encoder', model.clinical_encoder),
    ('occlusion_encoder', model.occlusion_encoder),
    ('context_fusion', model.context_fusion),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'vascular_stream_has_gradients_after_gate_activation': (
        gradient_norms['vascular_stream'] > 0
    ),
    'spatial_fusion_has_gradients_after_gate_activation': (
        gradient_norms['spatial_fusion'] > 0
    ),
    'clinical_encoder_has_gradients_after_gate_activation': (
        gradient_norms['clinical_encoder'] > 0
    ),
    'occlusion_encoder_has_gradients_after_gate_activation': (
        gradient_norms['occlusion_encoder'] > 0
    ),
    'context_fusion_has_gradients_after_gate_activation': (
        gradient_norms['context_fusion'] > 0
    ),
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'image_channels': int(image.shape[0]),
    'patch_size': PATCH_SIZE,
    'backbone_channels': [32, 64, 128, 224, 320],
    'baseline_architecture_preserved': True,
    'baseline_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    'graph_variant': 'two_layer_relation_aware_gated_GATv2',
    'graph_hidden': GRAPH_HIDDEN,
    'graph_heads': GRAPH_HEADS,
    'graph_layers': GRAPH_LAYERS,
    'relation_bases': RELATION_BASES,
    'graph_dropout': GRAPH_DROPOUT,
    'edge_weight_use': 'log_attention_prior',
    'territory_nodes': NUM_TERRITORY_NODES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'conditioning': (
        'preserved_bounded_zero_initialized_multiscale_FiLM_plus_'
        'confidence_gated_zero_initialized_vascular_context_fusion'
    ),
    'film_bound': FILM_BOUND,
    'fusion_bound': FUSION_BOUND,
    'asymmetric_vascular_stream': {
        'input_channels': [
            'cta', 'tmax', 'perfusion_valid',
            'cow_soft_union', 'lvo_proximity',
        ],
        'channels': VASCULAR_CHANNELS,
        'fusion_scales': ['quarter', 'eighth', 'sixteenth'],
        'initial_gate': 'exact_zero',
    },
    'clinical_token': {
        'input_dimension': CLINICAL_INPUT_DIM,
        'embedding_dimension': CLINICAL_EMBED_DIM,
        'values': 22,
        'paired_missingness_indicators': 22,
        'fold_specific_scaler_required_before_training': True,
    },
    'occlusion_source': {
        'value_dimension': OCCLUSION_VALUE_DIM,
        'embedding_dimension': OCCLUSION_EMBED_DIM,
        'validity_masked': True,
        'uses_tmax_not_ttp': True,
    },
    'confidence_features': [
        'perfusion_valid_fraction',
        'cow_prior_valid',
        'registration_confidence',
        'lvo_present',
    ],
    'staged_neutrality': {
        'new_fusion_gates_exactly_zero_at_initialization': True,
        'controlled_gate_activation_used_only_for_preflight': True,
        'saved_or_trained_weights_created': False,
    },
    'graph_to_logit_residual': False,
    'expert_router': False,
    'deep_supervision': True,
    'territory_auxiliary_head': True,
    'scaler_policy': 'fit_on_train_only_then_freeze',
    'expected_total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
        'asymmetric_vascular_stream': vascular_stream_parameters,
        'spatial_fusion': spatial_fusion_parameters,
        'clinical_encoder': clinical_parameters,
        'occlusion_encoder': occlusion_parameters,
        'context_fusion': context_fusion_parameters,
        'new_fusion_total': new_fusion_parameters,
        'preserved_v1_total': preserved_v1_parameters,
    },
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_sha256': scaler_sha256,
    'feature_availability_gate_path': str(AVAILABILITY_GATE_PATH),
    'architecture_input_contract_path': str(INPUT_CONTRACT_PATH),
    'ablation_differences': ablation_differences,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'model_training_authorized': False,
}
atomic_write_json(architecture_config, ARCHITECTURE_JSON_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **forward_checks,
    **ablation_checks,
    **gradient_checks,
    'feature_availability_audit_passed': (
        availability_gate.get('feature_availability_audit_passed') is True
    ),
    'input_contract_authorizes_architecture_implementation': (
        input_contract.get('architecture_implementation_authorized') is True
    ),
    'preserved_v1_parameter_count_matches': (
        preserved_v1_parameters == EXPECTED_BASELINE_PARAMETERS
    ),
    'parameter_count_in_frozen_efficiency_budget': (
        PARAMETER_BUDGET_MINIMUM
        <= total_parameters
        <= PARAMETER_BUDGET_MAXIMUM
    ),
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'new_fusion_parameter_count_is_bounded': (
        0 < new_fusion_parameters < 600_000
    ),
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'direct_graph_to_logit_residual_disabled': True,
    'expert_routing_disabled': True,
    'training_remained_locked': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'ablation', 'check': name, 'passed': passed}
    for name, passed in ablation_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
 ] + [
    {'category': 'final_gate', 'check': name, 'passed': passed}
    for name, passed in preflight_checks.items()
    if name not in {
        *manifest_checks.keys(),
        *scaler_checks.keys(),
        *forward_checks.keys(),
        *ablation_checks.keys(),
        *gradient_checks.keys(),
    }
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'ablation_differences': ablation_differences,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'controlled_gate_activation_was_diagnostic_only': True,
    'saved_or_trained_weights_created': False,
    'v2_loader_and_clinical_scaler_implementation_authorized': preflight_passed,
    'model_training_authorized': False,
}
atomic_write_json(gate_record, PREFLIGHT_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 MODEL CONSTRUCTION PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'New fusion parameters:              {new_fusion_parameters:,}')
print(f'  Asymmetric vascular stream:       {vascular_stream_parameters:,}')
print(f'  Spatial confidence fusion:        {spatial_fusion_parameters:,}')
print(f'  Clinical encoder:                 {clinical_parameters:,}')
print(f'  Occlusion encoder:                {occlusion_parameters:,}')
print(f'  Context fusion:                   {context_fusion_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('New fusion initially neutral:       True')
print('Controlled ablations responsive:    True')
print('Graph scaler fitted on:             train only (102 subjects)')
print('Clinical scaler fitted:             False (next preflight)')
print('Target arrays read:                 False')
print('Outcome workbooks read:              False')
print('Model training authorized:          False')
print(f'V2 loader implementation allowed:   {preflight_passed}')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print(
    '\nThe standalone block will now verify the completed multimodal-loader '
    'gate and continue into the isolated fold-0 pilot.'
)






PILOT_CODE_REVISION = 'r3_v2_1_isolated_static_hardened'
PILOT_VERSION = 'v2_1_fold0_v2_multimodal_ema_calibrated'
LOADER_PREFLIGHT_VERSION = 'v1_fold_clinical_scalers_multimodal_loader'
FIVEFOLD_PREFLIGHT_VERSION = 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_V2_PARAMETERS = 15_428_468
HELDOUT_FOLD = 0
PILOT_SEED = 20260928
PILOT_PATCH_SIZE = (128, 128, 64)
NUMBER_OF_FOLDS = 5
MINIMUM_POSITIVE_PATCHES_PER_BIN = 20
MAXIMUM_NEGATIVE_CENTER_ATTEMPTS = 512
MAXIMUM_CENTER_RETRIES = 16
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
PILOT_CLINICAL_FEATURE_NAMES = [
    'sex_female',
    'age_years',
    'atrial_fibrillation',
    'hypertension',
    'diabetes',
    'hyperlipidemia',
    'anticoagulation',
    'lipid_lowering_drugs',
    'platelet_aggregation_inhibitors',
    'glucose_mg_dl',
    'leucocytes_g_l',
    'crp_mg_dl',
    'inr',
    'wake_up_stroke',
    'in_house_stroke',
    'external_referral',
    'onset_to_door_hours',
    'alert_to_door_hours',
    'nihss_at_admission',
    'mrs_at_admission',
    'mrs_premorbid',
    'door_to_imaging_hours',
]

MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
LOADER_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / LOADER_PREFLIGHT_VERSION
)
LOADER_GATE_PATH = LOADER_PREFLIGHT_ROOT / 'v2_multimodal_loader_gate.json'
FIVEFOLD_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / FIVEFOLD_PREFLIGHT_VERSION
)
FIVEFOLD_GATE_PATH = (
    FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)
FIVEFOLD_POLICY_PATH = FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_training_policy.json'

PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
LEGACY_V2_PILOT_ROOT = (
    PROJECT_ROOT
    / 'checkpoints'
    / MODEL_VERSION
    / 'v1_fold0_v2_multimodal_ema_calibrated'
)
v21_write_roots = [OUTPUT_ROOT.resolve(), PILOT_ROOT.resolve()]
legacy_v2_roots = [
    LEGACY_V2_PREFLIGHT_ROOT.resolve(),
    LEGACY_V2_PILOT_ROOT.resolve(),
]
if set(v21_write_roots) & set(legacy_v2_roots):
    raise RuntimeError(
        'V2.1 output isolation failed: a write root overlaps the legacy V2 run.'
    )
if OUTPUT_ROOT.resolve() == PILOT_ROOT.resolve():
    raise RuntimeError('V2.1 preflight and pilot roots must be distinct.')
for write_root in v21_write_roots:
    assert_project_path_is_safe(write_root)
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_START_GATE_PATH = PILOT_ROOT / 'fold0_pilot_start_gate.json'
PILOT_CONFIG_PATH = PILOT_ROOT / 'fold0_pilot_config.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'fold0_training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'fold0_pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'fold0_pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'fold0_epoch1_schedule_audit.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_json_contract_hash(record, excluded_top_level_keys=()):
    excluded = set(excluded_top_level_keys)
    stable_record = {
        key: value for key, value in record.items() if key not in excluded
    }
    payload = json.dumps(
        to_builtin(stable_record),
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


pilot_required_paths = [
    MULTIMODAL_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    LOADER_GATE_PATH,
    FIVEFOLD_GATE_PATH,
    FIVEFOLD_POLICY_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    ARCHITECTURE_JSON_PATH,
    PREFLIGHT_GATE_PATH,
]
pilot_missing_paths = [str(path) for path in pilot_required_paths if not path.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Fold-0 pilot prerequisites are missing:\n' + '\n'.join(pilot_missing_paths)
    )

multimodal_manifest_sha256 = file_sha256(MULTIMODAL_MANIFEST_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)
loader_gate_sha256 = file_sha256(LOADER_GATE_PATH)
fivefold_gate_sha256 = file_sha256(FIVEFOLD_GATE_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)
v2_model_gate_sha256 = file_sha256(PREFLIGHT_GATE_PATH)
architecture_contract_sha256 = pilot_json_contract_hash(
    architecture_config,
    excluded_top_level_keys={'created_at_utc', 'ablation_differences'},
)
if multimodal_manifest_sha256 != EXPECTED_MULTIMODAL_MANIFEST_SHA256:
    raise RuntimeError('Multimodal candidate manifest hash changed.')
if fold_manifest_sha256 != EXPECTED_FOLD_MANIFEST_SHA256:
    raise RuntimeError('Five-fold manifest hash changed.')

loader_gate = json.loads(LOADER_GATE_PATH.read_text(encoding='utf-8'))
fivefold_gate = json.loads(FIVEFOLD_GATE_PATH.read_text(encoding='utf-8'))
fivefold_policy = json.loads(FIVEFOLD_POLICY_PATH.read_text(encoding='utf-8'))
v2_model_gate = json.loads(PREFLIGHT_GATE_PATH.read_text(encoding='utf-8'))
PILOT_ATLAS_DROPOUT_PROBABILITY = float(
    fivefold_policy['atlas_dropout_probability']
)
PILOT_GRAPH_FEATURE_MASK_PROBABILITY = float(
    fivefold_policy['graph_feature_mask_probability']
)
pilot_gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if torch.cuda.is_available()
    else 0.0
)
pilot_bfloat16_supported = bool(
    torch.cuda.is_available()
    and getattr(torch.cuda, 'is_bf16_supported', lambda: False)()
)

pilot_upstream_checks = {
    'v2_model_preflight_passed': v2_model_gate.get('preflight_passed') is True,
    'v2_model_parameter_count_matches': int(
        v2_model_gate.get('total_parameters', -1)
    ) == EXPECTED_V2_PARAMETERS,
    'v2_model_architecture_hash_matches': (
        v2_model_gate.get('architecture_config_sha256') == architecture_sha256
    ),
    'multimodal_loader_preflight_passed': (
        loader_gate.get('preflight_passed') is True
    ),
    'multimodal_manifest_hash_matches_loader_gate': (
        loader_gate.get('candidate_manifest_sha256')
        == multimodal_manifest_sha256
    ),
    'fold_manifest_hash_matches_loader_gate': (
        loader_gate.get('fold_manifest_sha256') == fold_manifest_sha256
    ),
    'graph_schema_hash_matches_loader_gate': (
        loader_gate.get('graph_schema_sha256') == safe_schema_sha256
    ),
    'pilot_implementation_authorized': (
        loader_gate.get('pilot_training_implementation_authorized') is True
    ),
    'upstream_training_remained_locked': (
        loader_gate.get('fold0_pilot_training_authorized') is False
        and loader_gate.get('full_fivefold_training_authorized') is False
    ),
    'loader_target_free': loader_gate.get('target_arrays_read') is False,
    'loader_outcome_free': loader_gate.get('outcome_workbooks_read') is False,
    'fivefold_loader_loss_sampler_passed': (
        fivefold_gate.get('preflight_passed') is True
        and fivefold_gate.get(
            'baseline_loader_loss_sampler_contract_verified'
        ) is True
    ),
    'fivefold_policy_training_locked': (
        fivefold_gate.get('fold0_pilot_training_authorized') is False
        and fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed_upstream': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed_upstream': fivefold_gate.get('test_targets_read') is False,
    'architecture_still_has_no_expert_router': (
        architecture_config.get('expert_router') is False
    ),
    'architecture_still_has_no_graph_logit_residual': (
        architecture_config.get('graph_to_logit_residual') is False
    ),
    'cuda_device_available_for_pilot': device.type == 'cuda',
    'pilot_gpu_has_at_least_20_gb': pilot_gpu_memory_gb >= 20.0,
    'pilot_gpu_supports_bfloat16': pilot_bfloat16_supported,
}
if not all(pilot_upstream_checks.values()):
    failed = [name for name, passed in pilot_upstream_checks.items() if not passed]
    raise RuntimeError('Fold-0 pilot upstream checks failed: ' + ', '.join(failed))

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].astype(str).tolist() != pilot_fold_df[
    'subject_id'
].astype(str).tolist():
    raise RuntimeError('Multimodal and fold manifests have different subjects.')
if 'development_fold' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_fold'] = pilot_fold_df[
        'development_fold'
    ].astype(int)
if 'development_role' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_role'] = pilot_fold_df[
        'development_role'
    ].astype(str)

pilot_required_columns = {
    'subject_id', 'dev_split', 'development_fold', 'vascufate_v2_npz_path',
    'trial8_npz_sha256', 'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
    'trial8_clinical_sidecar_sha256',
    'trial8_v2_multimodal_loader_preflight_passed',
    'trial8_model_training_authorized', 'trial8_supervision_mode',
    'trial8_effective_target_volume_ml',
    'trial8_effective_lesion_volume_bin', 'trial8_sampling_positive',
    'trial8_hard_metric_eligible',
}
pilot_missing_columns = sorted(
    pilot_required_columns - set(pilot_manifest_df.columns)
)
if pilot_missing_columns:
    raise RuntimeError(
        'Multimodal manifest is missing columns: ' + ', '.join(pilot_missing_columns)
    )

development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'].astype(int) != HELDOUT_FOLD
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'].astype(int) == HELDOUT_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()

pilot_manifest_checks = {
    'manifest_has_148_subjects': len(pilot_manifest_df) == 148,
    'manifest_subjects_unique': pilot_manifest_df['subject_id'].nunique() == 148,
    'split_counts_match': pilot_manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'loader_preflight_passed_for_all': bool(
        pilot_manifest_df[
            'trial8_v2_multimodal_loader_preflight_passed'
        ].map(as_bool).all()
    ),
    'manifest_training_remained_locked': not bool(
        pilot_manifest_df['trial8_model_training_authorized'].map(as_bool).any()
    ),
    'fold0_fit_has_82_subjects': len(fold0_fit_df) == 82,
    'fold0_heldout_has_20_subjects': len(fold0_heldout_df) == 20,
    'locked_validation_has_23_subjects': len(locked_validation_df) == 23,
    'sealed_test_has_23_subjects': len(sealed_test_df) == 23,
    'fold0_fit_and_heldout_are_disjoint': not bool(
        set(fold0_fit_df['subject_id']) & set(fold0_heldout_df['subject_id'])
    ),
}
if not all(pilot_manifest_checks.values()):
    failed = [name for name, passed in pilot_manifest_checks.items() if not passed]
    raise RuntimeError('Fold-0 manifest checks failed: ' + ', '.join(failed))

fold0_graph_scaler_path = Path(loader_gate['graph_scaler_paths'][str(HELDOUT_FOLD)])
fold0_graph_scaler_sha256 = loader_gate['graph_scaler_sha256'][
    str(HELDOUT_FOLD)
]
fold0_clinical_scaler_path = Path(
    loader_gate['clinical_scaler_paths'][str(HELDOUT_FOLD)]
)
fold0_clinical_scaler_sha256 = loader_gate['clinical_scaler_sha256'][
    str(HELDOUT_FOLD)
]
for scaler_path, expected_hash, label in [
    (fold0_graph_scaler_path, fold0_graph_scaler_sha256, 'graph'),
    (fold0_clinical_scaler_path, fold0_clinical_scaler_sha256, 'clinical'),
]:
    if not scaler_path.exists():
        raise FileNotFoundError(f'Fold-0 {label} scaler is missing: {scaler_path}')
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-0 {label} scaler hash changed.')

with np.load(fold0_graph_scaler_path, allow_pickle=False) as data:
    fold0_graph_scaler = {name: np.asarray(data[name]) for name in data.files}
with np.load(fold0_clinical_scaler_path, allow_pickle=False) as data:
    fold0_clinical_scaler = {name: np.asarray(data[name]) for name in data.files}

if decode_strings(fold0_graph_scaler['territory_feature_names']) != (
    EXPECTED_NODE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 territory feature order changed.')
if decode_strings(fold0_graph_scaler['vessel_feature_names']) != (
    MODEL_VESSEL_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 vessel feature order changed.')
if decode_strings(fold0_graph_scaler['case_feature_names']) != (
    MODEL_CASE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 case feature order changed.')
if decode_strings(
    fold0_clinical_scaler['feature_names']
) != PILOT_CLINICAL_FEATURE_NAMES:
    raise RuntimeError('Fold-0 clinical feature order changed.')

fold0_fit_subject_hash = pilot_subject_hash(fold0_fit_df['subject_id'])
fold0_heldout_subject_hash = pilot_subject_hash(fold0_heldout_df['subject_id'])
pilot_scaler_contract_checks = {
    'graph_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_graph_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'graph_scaler_fit_count_is_82': int(
        np.asarray(fold0_graph_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'graph_scaler_fit_subject_hash_matches': scalar_string(
        fold0_graph_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'graph_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_graph_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'clinical_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_clinical_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'clinical_scaler_fit_count_is_82': int(
        np.asarray(fold0_clinical_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'clinical_scaler_heldout_count_is_20': int(
        np.asarray(fold0_clinical_scaler['heldout_subject_count']).reshape(-1)[0]
    ) == 20,
    'clinical_scaler_fit_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'clinical_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'graph_scaler_is_target_free': not bool(
        np.asarray(fold0_graph_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_target_free': not bool(
        np.asarray(fold0_clinical_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_outcome_free': not bool(
        np.asarray(
            fold0_clinical_scaler['outcome_workbooks_read']
        ).reshape(-1)[0]
    ),
}
if not all(pilot_scaler_contract_checks.values()):
    failed = [
        name for name, passed in pilot_scaler_contract_checks.items()
        if not passed
    ]
    raise RuntimeError('Fold-0 scaler contract failed: ' + ', '.join(failed))

print('\n' + '=' * 80)
print('TRIAL8 V2.1 FOLD-0 PILOT CONTRACT PASSED')
print('=' * 80)
print(f'Pilot code revision:                 {PILOT_CODE_REVISION}')
print(f'Isolated construction root:          {OUTPUT_ROOT}')
print(f'Isolated pilot root:                 {PILOT_ROOT}')
print(f'Pilot GPU memory:                    {pilot_gpu_memory_gb:.2f} GB')
print(f'Fold-0 fit / held-out:               {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
print(f'Locked validation / sealed test:     {len(locked_validation_df)} / {len(sealed_test_df)}')
print('Target access authorized:            development train split only')
print('Locked validation targets read:      False')
print('Test targets read:                   False')


for disposable_name in [
    'model', 'batch', 'normal_output', 'atlas_dropped_output',
    'new_sources_dropped_output', 'activated_output',
    'vascular_ablation_output', 'occlusion_ablation_output',
    'clinical_ablation_output', 'gradient_output', 'smoke_loss',
    'module', 'parameter',
    'image', 'region_map', 'cow_soft_union', 'lvo_proximity',
    'image_patch', 'region_patch', 'cow_patch', 'lvo_patch',
    'vascular_patch', 'clinical_input', 'occlusion_values',
    'occlusion_valid', 'confidence_features', 'node_features',
    'node_feature_valid', 'node_valid', 'node_reliability',
    'vessel_features_source', 'vessel_feature_valid_source',
    'vessel_valid', 'case_features_source', 'scaled_territory',
    'scaled_vessel', 'scaled_case', 'selected_vessel',
    'selected_vessel_valid', 'selected_case',
]:
    globals().pop(disposable_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(value) for value in mask.shape),
        'packed': np.packbits(mask.reshape(-1)),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    return np.unpackbits(record['packed'], count=count).reshape(
        record['shape']
    ).astype(bool)


fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')
all_development_row_by_subject = {
    str(row['subject_id']): row for row in development_df.to_dict('records')
}
pilot_mask_records = {}
pilot_shape_by_subject = {}
pilot_target_rows = []
for row in development_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required_target_arrays = {
            'target_binary', 'target_soft', 'binary_loss_weight',
            'crop_mask', 'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required_target_arrays - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        hard_weight = pilot_scalar(data, 'binary_loss_weight')
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
        source_volume_ml = pilot_scalar(data, 'source_target_volume_ml')
    if target_binary.shape != target_soft.shape or target_binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if not np.isfinite(target_soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target arrays are non-finite.')
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        sampling_support = target_binary
        expected_hard_weight = 1.0
    elif supervision_mode == 'soft_only_subvoxel':
        sampling_support = target_soft > 0
        expected_hard_weight = 0.0
    elif supervision_mode == 'true_empty':
        sampling_support = np.zeros_like(target_binary)
        expected_hard_weight = 1.0
    else:
        raise RuntimeError(f'{subject_id}: invalid supervision mode.')
    exclusion_support = target_binary | (target_soft > 0)
    pilot_shape_by_subject[subject_id] = tuple(
        int(value) for value in target_binary.shape
    )
    if not math.isclose(hard_weight, expected_hard_weight):
        raise RuntimeError(f'{subject_id}: hard-loss policy changed.')
    if subject_id in fit_row_by_subject:
        pilot_mask_records[subject_id] = {
            'sampling_support': pilot_pack_mask(sampling_support),
            'exclusion_support': pilot_pack_mask(exclusion_support),
            'crop_mask': pilot_pack_mask(crop_mask),
        }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_fold': int(row['development_fold']),
        'pilot_role': 'fit' if subject_id in fit_row_by_subject else 'heldout',
        'supervision_mode': supervision_mode,
        'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
        'sampling_support_voxels': int(sampling_support.sum()),
        'binary_voxels': int(target_binary.sum()),
        'soft_nonzero_voxels': int((target_soft > 0).sum()),
        'source_target_volume_ml': source_volume_ml,
        'hard_weight': hard_weight,
        'voxel_volume_ml': float(np.prod(spacing) / 1000.0),
        'target_arrays_read_split': 'train',
    })
pilot_target_df = pd.DataFrame(pilot_target_rows)
atomic_write_csv(pilot_target_df, PILOT_TARGET_AUDIT_PATH)
pilot_target_by_subject = pilot_target_df.set_index('subject_id').to_dict('index')

pilot_target_checks = {
    'exactly_102_development_targets_read': len(pilot_target_df) == 102,
    'target_access_roles_are_fit_or_heldout': set(
        pilot_target_df['pilot_role']
    ) == {'fit', 'heldout'},
    'target_access_is_train_split_only': set(
        pilot_target_df['target_arrays_read_split']
    ) == {'train'},
    'fit_sampling_masks_cover_82_subjects': len(pilot_mask_records) == 82,
    'soft_only_hard_loss_disabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'] == 'soft_only_subvoxel',
            'hard_weight',
        ].eq(0.0).all()
    ),
    'hard_and_empty_hard_loss_enabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'].isin(
                ['hard_and_soft', 'true_empty']
            ),
            'hard_weight',
        ].eq(1.0).all()
    ),
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
}
if not all(pilot_target_checks.values()):
    failed = [name for name, passed in pilot_target_checks.items() if not passed]
    raise RuntimeError('Fold-0 target checks failed: ' + ', '.join(failed))
for disposable_name in [
    'target_binary', 'target_soft', 'crop_mask', 'sampling_support',
    'exclusion_support', 'spacing',
]:
    globals().pop(disposable_name, None)
gc.collect()

territory_edge_index_numpy = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type_numpy = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)
PILOT_RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def pilot_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, PILOT_PATCH_SIZE):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def pilot_extract_patch(array, slices, spatial_axes):
    source = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source[axis] = spatial_slice
    cropped = array[tuple(source)]
    return center_crop_or_pad(cropped, PILOT_PATCH_SIZE, spatial_axes)


region_cache = {}
region_cache_order = []
REGION_CACHE_LIMIT = 24


def pilot_load_region_map(subject_id):
    if subject_id in region_cache:
        return region_cache[subject_id]
    row = all_development_row_by_subject[subject_id]
    shape = pilot_shape_by_subject[subject_id]
    if not as_bool(row['trial8_atlas_enabled']):
        region = np.zeros(shape, dtype=np.uint8)
    else:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region = np.asanyarray(atlas_image.dataobj)
        if region.ndim == 4 and region.shape[-1] == 1:
            region = region[..., 0]
        region = np.rint(region).astype(np.uint8)
        if region.shape != shape:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region.min() < 0 or region.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    region_cache[subject_id] = region
    region_cache_order.append(subject_id)
    if len(region_cache_order) > REGION_CACHE_LIMIT:
        evicted = region_cache_order.pop(0)
        region_cache.pop(evicted, None)
    return region


def pilot_topology_candidates(support, region_map):
    lesion_labels = set(
        int(value) for value in np.unique(region_map[support])
        if 1 <= int(value) <= 30
    )
    relation_priority = {2: 0, 1: 1, 3: 2}
    relation_by_label = {}
    for edge_index in range(territory_edge_index_numpy.shape[1]):
        source = int(territory_edge_index_numpy[0, edge_index]) + 1
        destination = int(territory_edge_index_numpy[1, edge_index]) + 1
        relation = int(territory_edge_type_numpy[edge_index])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in relation_priority
        ):
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    labels = sorted(
        relation_by_label,
        key=lambda label: (relation_priority[relation_by_label[label]], label),
    )
    return labels, relation_by_label


def pilot_sample_coordinates(mask, rng):
    flat = np.flatnonzero(mask)
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > MAXIMUM_NEGATIVE_CENTER_ATTEMPTS:
        flat = rng.choice(
            flat, size=MAXIMUM_NEGATIVE_CENTER_ATTEMPTS, replace=False
        )
    else:
        flat = flat.copy()
        rng.shuffle(flat)
    return np.column_stack(np.unravel_index(flat, mask.shape)).astype(np.int64)


def pilot_zero_support_center(exclusion, candidate, rng):
    for center in pilot_sample_coordinates(candidate, rng):
        slices = pilot_patch_slices(exclusion.shape, center)
        if not exclusion[slices].any():
            return center, slices
    return None, None


def pilot_positive_center(subject_id, rng, epoch):
    support = pilot_unpack_mask(
        pilot_mask_records[subject_id]['sampling_support']
    )
    if not support.any():
        raise RuntimeError(f'{subject_id}: positive schedule has no support.')
    if pilot_target_by_subject[subject_id]['supervision_mode'] == (
        'soft_only_subvoxel'
    ):
        labels, component_count = ndimage.label(
            support, structure=np.ones((3, 3, 3), dtype=np.uint8)
        )
        selected_component = 1 + ((epoch - 1) % component_count)
        coordinates = np.argwhere(labels == selected_component)
    else:
        coordinates = np.argwhere(support)
    seed = coordinates[int(rng.integers(0, len(coordinates)))].astype(np.int64)
    for _ in range(MAXIMUM_CENTER_RETRIES):
        jitter = np.asarray([
            int(rng.integers(-12, 13)),
            int(rng.integers(-12, 13)),
            int(rng.integers(-4, 5)),
        ])
        center = np.clip(seed + jitter, 0, np.asarray(support.shape) - 1)
        slices = pilot_patch_slices(support.shape, center)
        if support[slices].any():
            return center, slices
    slices = pilot_patch_slices(support.shape, seed)
    if not support[slices].any():
        raise RuntimeError(f'{subject_id}: failed to retain positive support.')
    return seed, slices


def pilot_balanced_positive_schedule(rng):
    schedule = []
    for lesion_bin in POSITIVE_BINS:
        pool = fold0_fit_df.loc[
            (fold0_fit_df['trial8_effective_lesion_volume_bin'] == lesion_bin)
            & fold0_fit_df['trial8_sampling_positive'].map(as_bool),
            'subject_id',
        ].astype(str).tolist()
        if not pool:
            raise RuntimeError(f'Fold-0 positive bin {lesion_bin} is empty.')
        target_count = max(MINIMUM_POSITIVE_PATCHES_PER_BIN, len(pool))
        selected = pool.copy()
        hard_pool = [
            subject_id for subject_id in pool
            if pilot_target_by_subject[subject_id]['supervision_mode']
            == 'hard_and_soft'
        ]
        rng.shuffle(hard_pool)
        if not hard_pool and len(selected) < target_count:
            raise RuntimeError(f'No hard-target repeats for {lesion_bin}.')
        repeat_index = 0
        while len(selected) < target_count:
            selected.append(hard_pool[repeat_index % len(hard_pool)])
            repeat_index += 1
        rng.shuffle(selected)
        schedule.extend(selected)
    rng.shuffle(schedule)
    return schedule


def pilot_build_epoch_schedule(epoch):
    rng = np.random.default_rng(PILOT_SEED + epoch * 1_000_003)
    positive_subjects = pilot_balanced_positive_schedule(rng)
    positive_count = len(positive_subjects)
    topology_count = int(math.ceil(positive_count / 4.0))
    brain_count = int(math.floor(positive_count / 4.0))
    topology_pool = fold0_fit_df.loc[
        fold0_fit_df['trial8_atlas_enabled'].map(as_bool)
        & fold0_fit_df['trial8_sampling_positive'].map(as_bool)
        & fold0_fit_df['trial8_supervision_mode'].eq('hard_and_soft'),
        'subject_id',
    ].astype(str).tolist()
    brain_pool = fold0_fit_df.loc[
        ~fold0_fit_df['trial8_supervision_mode'].eq('soft_only_subvoxel'),
        'subject_id',
    ].astype(str).tolist()
    entries = []
    for subject_id in positive_subjects:
        center, _ = pilot_positive_center(subject_id, rng, epoch)
        entries.append({
            'sample_type': 'positive',
            'subject_id': subject_id,
            'center': center,
            'topology_relation': '',
        })
    for _ in range(topology_count):
        selected = None
        start = int(rng.integers(0, len(topology_pool)))
        for attempt in range(len(topology_pool)):
            subject_id = topology_pool[(start + attempt) % len(topology_pool)]
            support = pilot_unpack_mask(
                pilot_mask_records[subject_id]['sampling_support']
            )
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            region = pilot_load_region_map(subject_id)
            labels, relation_by_label = pilot_topology_candidates(support, region)
            for relation in [2, 1, 3]:
                relation_labels = [
                    label for label in labels
                    if relation_by_label[label] == relation
                ]
                if not relation_labels:
                    continue
                center, _ = pilot_zero_support_center(
                    exclusion, np.isin(region, relation_labels) & crop_mask, rng
                )
                if center is not None:
                    selected = {
                        'sample_type': 'topology_negative',
                        'subject_id': subject_id,
                        'center': center,
                        'topology_relation': PILOT_RELATION_NAME[relation],
                    }
                    break
            if selected is not None:
                break
        if selected is None:
            raise RuntimeError('Could not construct a topology-negative patch.')
        entries.append(selected)
    for _ in range(brain_count):
        selected = None
        start = int(rng.integers(0, len(brain_pool)))
        for attempt in range(len(brain_pool)):
            subject_id = brain_pool[(start + attempt) % len(brain_pool)]
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            center, _ = pilot_zero_support_center(exclusion, crop_mask, rng)
            if center is not None:
                selected = {
                    'sample_type': 'brain_negative',
                    'subject_id': subject_id,
                    'center': center,
                    'topology_relation': '',
                }
                break
        if selected is None:
            raise RuntimeError('Could not construct a brain-negative patch.')
        entries.append(selected)
    rng.shuffle(entries)
    return entries






graph_input_cache = {}
clinical_input_cache = {}


def pilot_load_graph_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in graph_input_cache:
        return graph_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: graph sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    territory_mean_local = np.asarray(
        fold0_graph_scaler['territory_mean'], dtype=np.float32
    )
    territory_std_local = np.asarray(
        fold0_graph_scaler['territory_std'], dtype=np.float32
    )
    vessel_indices = np.asarray(
        fold0_graph_scaler['vessel_source_indices'], dtype=np.int64
    )
    case_indices = np.asarray(
        fold0_graph_scaler['case_source_indices'], dtype=np.int64
    )
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_indices] & vessel_presence[:, None]
    )
    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean_local[None],
        territory_std_local[None],
    )
    vessel_scaled = masked_scale(
        vessel_source[:, vessel_indices],
        vessel_feature_valid,
        np.asarray(fold0_graph_scaler['vessel_mean'], dtype=np.float32)[None],
        np.asarray(fold0_graph_scaler['vessel_std'], dtype=np.float32)[None],
    )
    case_selected = case_source[case_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scale_mask_local = np.asarray(
        fold0_graph_scaler['case_scale_mask']
    ) > 0
    case_mean_local = np.asarray(
        fold0_graph_scaler['case_mean'], dtype=np.float32
    )
    case_std_local = np.asarray(
        fold0_graph_scaler['case_std'], dtype=np.float32
    )
    case_scaled[case_scale_mask_local] = np.clip(
        (
            case_scaled[case_scale_mask_local]
            - case_mean_local[case_scale_mask_local]
        ) / case_std_local[case_scale_mask_local],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean',
        'minimum_lvo_distance_mm', 'lvo_overlap_fraction',
    ]
    occlusion_graph_values = []
    occlusion_graph_valid = []
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = vessel_feature_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_graph_values.append(float(
                vessel_scaled[valid_nodes, feature_index].mean()
            ))
            occlusion_graph_valid.append(1.0)
        else:
            occlusion_graph_values.append(0.0)
            occlusion_graph_valid.append(0.0)
    output = {
        'territory_features': territory_scaled.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': vessel_scaled.astype(np.float32),
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence.astype(bool),
        'case_features': case_scaled.astype(np.float32),
        'cow_prior_valid': bool(
            case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
        ),
        'registration_confidence': float(
            case_source[
                SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
            ]
        ),
        'occlusion_graph_values': np.asarray(
            occlusion_graph_values, dtype=np.float32
        ),
        'occlusion_graph_valid': np.asarray(
            occlusion_graph_valid, dtype=np.float32
        ),
    }
    if not all(
        np.isfinite(value).all()
        for value in output.values()
        if np.asarray(value).dtype.kind in {'f', 'c'}
    ):
        raise RuntimeError(f'{subject_id}: scaled graph input is non-finite.')
    graph_input_cache[subject_id] = output
    return output


def pilot_load_clinical_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in clinical_input_cache:
        return clinical_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_clinical_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: clinical sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        names = decode_strings(data['feature_names'])
    scaler_names = decode_strings(fold0_clinical_scaler['feature_names'])
    if names != scaler_names or values.shape != (22,) or valid.shape != (22,):
        raise RuntimeError(f'{subject_id}: clinical schema changed.')
    imputed = values.copy()
    imputation = np.asarray(
        fold0_clinical_scaler['imputation'], dtype=np.float32
    )
    imputed[~valid] = imputation[~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(fold0_clinical_scaler['scale_mask']) > 0
    clinical_mean = np.asarray(fold0_clinical_scaler['mean'], dtype=np.float32)
    clinical_std = np.asarray(fold0_clinical_scaler['std'], dtype=np.float32)
    clinical_clip = float(
        np.asarray(fold0_clinical_scaler['clip_value']).reshape(-1)[0]
    )
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - clinical_mean[scale_mask])
        / clinical_std[scale_mask],
        -clinical_clip,
        clinical_clip,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    output = {
        'clinical_input': token,
        'clinical_available': np.asarray(valid.any(), dtype=bool),
    }
    clinical_input_cache[subject_id] = output
    return output


def pilot_model_input_from_patches(
    row,
    image_patch,
    region_patch,
    cow_patch,
    lvo_patch,
    lvo_present,
    apply_graph_mask=False,
    rng=None,
):
    graph = {
        key: np.asarray(value).copy()
        for key, value in pilot_load_graph_input(row).items()
    }
    clinical = pilot_load_clinical_input(row)
    if apply_graph_mask and rng is not None:
        mask = (
            rng.random(graph['territory_features'].shape)
            < PILOT_GRAPH_FEATURE_MASK_PROBABILITY
        ) & (graph['territory_feature_valid'] > 0)
        graph['territory_features'][mask] = 0.0
        graph['territory_feature_valid'][mask] = 0.0
    input_channels = EXPECTED_INPUT_CHANNELS
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch,
        lvo_patch,
    ]).astype(np.float32)
    confidence = np.asarray([
        float(vascular_patch[2].mean()),
        float(graph['cow_prior_valid']),
        float(np.clip(graph['registration_confidence'], 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_values = np.concatenate([
        np.asarray([float(lvo_present), float(lvo_patch.max())], dtype=np.float32),
        graph['occlusion_graph_values'],
    ]).astype(np.float32)
    occlusion_valid = np.concatenate([
        np.asarray([1.0, float(lvo_present)], dtype=np.float32),
        graph['occlusion_graph_valid'],
    ]).astype(np.float32)
    return {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': graph['territory_features'].astype(np.float32),
        'territory_feature_valid': graph[
            'territory_feature_valid'
        ].astype(np.float32),
        'territory_valid': graph['territory_valid'].astype(bool),
        'territory_reliability': graph['territory_reliability'].astype(np.float32),
        'vessel_features': graph['vessel_features'].astype(np.float32),
        'vessel_feature_valid': graph['vessel_feature_valid'].astype(np.float32),
        'vessel_valid': graph['vessel_valid'].astype(bool),
        'case_features': graph['case_features'].astype(np.float32),
        'atlas_enabled': np.asarray(
            as_bool(row['trial8_atlas_enabled']), dtype=bool
        ),
        'vascular_image': vascular_patch,
        'confidence_features': confidence,
        'occlusion_values': occlusion_values,
        'occlusion_valid': occlusion_valid,
        'clinical_input': clinical['clinical_input'].astype(np.float32),
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': clinical['clinical_available'],
    }


def pilot_intensity_augment(image, rng):
    output = np.asarray(image, dtype=np.float32).copy()
    intensity_channels = min(13, output.shape[0])
    gains = rng.uniform(0.92, 1.08, size=(intensity_channels, 1, 1, 1))
    offsets = rng.uniform(-0.02, 0.02, size=(intensity_channels, 1, 1, 1))
    output[:intensity_channels] = output[:intensity_channels] * gains + offsets
    if rng.random() < 0.50:
        noise_std = float(rng.uniform(0.0, 0.015))
        output[:intensity_channels] += rng.normal(
            0.0, noise_std, size=output[:intensity_channels].shape
        ).astype(np.float32)
    if rng.random() < 0.40:
        gamma = float(rng.uniform(0.90, 1.10))
        nonnegative_end = min(9, intensity_channels)
        output[:nonnegative_end] = np.power(
            np.clip(output[:nonnegative_end], 0.0, 1.0), gamma
        )
    output[:min(9, intensity_channels)] = np.clip(
        output[:min(9, intensity_channels)], 0.0, 1.0
    )
    if intensity_channels > 9:
        output[9:intensity_channels] = np.clip(
            output[9:intensity_channels], -1.0, 1.0
        )
    return output.astype(np.float32)


class Trial8Fold0PatchDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.counts = {}
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)
        self.counts = {
            name: sum(entry['sample_type'] == name for entry in self.schedule)
            for name in ['positive', 'topology_negative', 'brain_negative']
        }

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        with np.load(npz_path, allow_pickle=False) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            input_channels = decode_strings(data['input_channels'])
            target_binary = np.asarray(data['target_binary'], dtype=np.float32)
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
            lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
            lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            hard_weight = pilot_scalar(data, 'binary_loss_weight')
        if input_channels != EXPECTED_INPUT_CHANNELS:
            raise RuntimeError(f'{subject_id}: image channel order changed.')
        slices = pilot_patch_slices(target_binary.shape, entry['center'])
        image_patch = pilot_extract_patch(image, slices, (1, 2, 3))
        binary_patch = pilot_extract_patch(target_binary, slices, (0, 1, 2))
        soft_patch = pilot_extract_patch(target_soft, slices, (0, 1, 2))
        cow_patch = pilot_extract_patch(cow, slices, (0, 1, 2))
        lvo_patch = pilot_extract_patch(lvo, slices, (0, 1, 2))
        region_patch = pilot_extract_patch(
            pilot_load_region_map(subject_id), slices, (0, 1, 2)
        )
        if entry['sample_type'] == 'positive' and not bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: positive patch lost target support.')
        if entry['sample_type'] != 'positive' and bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: negative patch contains target support.')
        image_patch = pilot_intensity_augment(image_patch, rng)
        model_inputs = pilot_model_input_from_patches(
            row,
            image_patch,
            region_patch,
            cow_patch,
            lvo_patch,
            lvo_present,
            apply_graph_mask=True,
            rng=rng,
        )
        model_inputs.update({
            'target_binary': binary_patch[None].astype(np.float32),
            'target_soft': soft_patch[None].astype(np.float32),
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'supervision_mode': str(
                pilot_target_by_subject[subject_id]['supervision_mode']
            ),
        })
        return model_inputs


MODEL_INPUT_KEYS = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available',
]


def pilot_to_device(cpu_batch):
    output = {}
    for key, value in cpu_batch.items():
        if torch.is_tensor(value):
            output[key] = value.to(device, non_blocking=True)
        else:
            output[key] = value
    return output


def pilot_spatial_augment(batch):
    batch_size = int(batch['image'].shape[0])
    if float(torch.rand(1).item()) >= 0.60:
        batch['spatially_augmented_fraction'] = torch.tensor(
            0.0, device=batch['image'].device
        )
        return batch
    positive = [str(value) == 'positive' for value in batch['sample_type']]
    supervision_modes = [
        str(value) for value in batch['supervision_mode']
    ]
    batch_device = batch['image'].device
    for _ in range(4):
        angles = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * math.radians(8.0)
        scales = 0.92 + torch.rand(batch_size, device=batch_device) * 0.16
        cosine = torch.cos(angles) * scales
        sine = torch.sin(angles) * scales
        theta = torch.zeros(
            batch_size, 3, 4, device=batch_device, dtype=torch.float32
        )
        
        
        
        theta[:, 0, 0] = scales
        theta[:, 1, 1] = cosine
        theta[:, 1, 2] = -sine
        theta[:, 2, 1] = sine
        theta[:, 2, 2] = cosine
        theta[:, 0, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.025
        theta[:, 1, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        theta[:, 2, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        grid = F.affine_grid(theta, batch['image'].shape, align_corners=False)

        transformed = dict(batch)
        transformed_image = torch.empty_like(batch['image'])
        transformed_image[:, :13] = F.grid_sample(
            batch['image'][:, :13], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed_image[:, 13:] = F.grid_sample(
            batch['image'][:, 13:], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['image'] = transformed_image
        transformed_vascular = torch.empty_like(batch['vascular_image'])
        continuous_indices = [0, 1, 3, 4]
        transformed_vascular[:, continuous_indices] = F.grid_sample(
            batch['vascular_image'][:, continuous_indices], grid,
            mode='bilinear', padding_mode='zeros', align_corners=False,
        )
        transformed_vascular[:, 2:3] = F.grid_sample(
            batch['vascular_image'][:, 2:3], grid,
            mode='nearest', padding_mode='zeros', align_corners=False,
        )
        transformed['vascular_image'] = transformed_vascular
        transformed['target_soft'] = F.grid_sample(
            batch['target_soft'], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed['target_binary'] = F.grid_sample(
            batch['target_binary'], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['region_map'] = F.grid_sample(
            batch['region_map'][:, None].float(), grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )[:, 0].long()
        positive_support_retained = True
        for batch_index, is_positive in enumerate(positive):
            if not is_positive:
                continue
            soft_retained = bool(
                (transformed['target_soft'][batch_index] > 0).any().item()
            )
            hard_retained = bool(
                (transformed['target_binary'][batch_index] > 0).any().item()
            )
            mode = supervision_modes[batch_index]
            if not soft_retained or (
                mode == 'hard_and_soft' and not hard_retained
            ):
                positive_support_retained = False
                break
        if not positive_support_retained:
            continue
        transformed['confidence_features'] = batch[
            'confidence_features'
        ].clone()
        transformed['confidence_features'][:, 0] = transformed_vascular[
            :, 2
        ].float().mean(dim=(1, 2, 3))
        transformed['occlusion_values'] = batch['occlusion_values'].clone()
        transformed['occlusion_values'][:, 1] = transformed_vascular[
            :, 4
        ].float().amax(dim=(1, 2, 3))
        transformed['spatially_augmented_fraction'] = torch.tensor(
            1.0, device=batch_device
        )
        return transformed
    batch['spatially_augmented_fraction'] = torch.tensor(
        0.0, device=batch_device
    )
    return batch






def pilot_soft_dice_per_case(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_soft_focal_bce_per_case(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, raw.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_per_case(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error_floor = 1e-6
    return (
        (1.0 - tversky).clamp_min(error_floor).pow(gamma)
        - float(error_floor ** gamma)
    )


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    dice = pilot_soft_dice_per_case(logits, soft_target)
    focal_bce = pilot_soft_focal_bce_per_case(logits, soft_target)
    hard_tversky = pilot_focal_tversky_per_case(logits, binary_target)
    hard_contribution = 0.20 * hard_tversky * hard_weight.reshape(-1)
    total = 0.50 * dice + 0.30 * focal_bce + hard_contribution
    return total.mean(), {
        'soft_dice_loss': dice.mean(),
        'soft_focal_bce_loss': focal_bce.mean(),
        'hard_focal_tversky_loss': hard_tversky.mean(),
        'hard_contribution': hard_contribution.mean(),
    }


def pilot_node_loss(output, batch):
    node_logits = output['territory_node_logits'].float()
    targets = torch.zeros_like(node_logits)
    valid = (
        batch['territory_valid'].bool()
        & batch['atlas_enabled'][:, None].bool()
        & (batch['hard_weight'][:, None] > 0)
    )
    for batch_index in range(node_logits.shape[0]):
        labels = batch['region_map'][batch_index]
        lesion = batch['target_binary'][batch_index, 0] > 0.5
        if bool(lesion.any().item()):
            present_labels = torch.unique(labels[lesion])
            present_labels = present_labels[
                (present_labels >= 1) & (present_labels <= 30)
            ]
            if len(present_labels) > 0:
                targets[batch_index, present_labels.long() - 1] = 1.0
    if not bool(valid.any().item()):
        return node_logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        node_logits,
        targets,
        reduction='none',
        pos_weight=torch.full(
            (30,), 4.0, device=node_logits.device, dtype=node_logits.dtype
        ),
    )
    return raw[valid].mean()


def pilot_compute_training_loss(study_model, cpu_batch):
    batch = pilot_to_device(cpu_batch)
    batch = pilot_spatial_augment(batch)
    output = study_model(**{key: batch[key] for key in MODEL_INPUT_KEYS})
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    auxiliary_eligible = (batch['hard_weight'] > 0).float()
    half_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], size=output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'],
        size=output['auxiliary_quarter_logits'].shape[2:], mode='nearest',
    )
    half_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_half_logits'], half_binary
        ) * batch['hard_weight']
    )
    quarter_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_quarter_logits'], quarter_binary
        ) * batch['hard_weight']
    )
    denominator = auxiliary_eligible.sum().clamp_min(1.0)
    half_loss = (half_per_case * auxiliary_eligible).sum() / denominator
    quarter_loss = (quarter_per_case * auxiliary_eligible).sum() / denominator
    node_loss = pilot_node_loss(output, batch)
    total = main_loss + 0.15 * half_loss + 0.10 * quarter_loss + 0.02 * node_loss
    parts = {
        'total_loss': total,
        'main_loss': main_loss,
        **main_parts,
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
        'vascular_retained_fraction': output[
            'effective_vascular_enabled'
        ].float().mean(),
        'occlusion_retained_fraction': output[
            'effective_occlusion_enabled'
        ].float().mean(),
        'clinical_retained_fraction': output[
            'effective_clinical_enabled'
        ].float().mean(),
        'context_gate_absolute_mean': output[
            'context_fusion_gate'
        ].float().abs().mean(),
        'quarter_gate_absolute_mean': output[
            'vascular_quarter_gate'
        ].float().abs().mean(),
        'eighth_gate_absolute_mean': output[
            'vascular_eighth_gate'
        ].float().abs().mean(),
        'bottleneck_gate_absolute_mean': output[
            'vascular_bottleneck_gate'
        ].float().abs().mean(),
        'spatially_augmented_fraction': batch[
            'spatially_augmented_fraction'
        ],
    }
    return total, parts


def pilot_sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_extract_window(array, starts, spatial_axes):
    output_shape = list(array.shape)
    for axis, patch in zip(spatial_axes, PILOT_PATCH_SIZE):
        output_shape[axis] = patch
    output = np.zeros(output_shape, dtype=array.dtype)
    source = [slice(None)] * array.ndim
    destination = [slice(None)] * array.ndim
    source_spatial = []
    destination_spatial = []
    for axis, start, patch in zip(spatial_axes, starts, PILOT_PATCH_SIZE):
        stop = min(start + patch, array.shape[axis])
        copied = stop - start
        destination_start = max((patch - copied) // 2, 0)
        source[axis] = slice(start, stop)
        destination[axis] = slice(destination_start, destination_start + copied)
        source_spatial.append(slice(start, stop))
        destination_spatial.append(
            slice(destination_start, destination_start + copied)
        )
    output[tuple(destination)] = array[tuple(source)]
    return output, tuple(source_spatial), tuple(destination_spatial)


def pilot_gaussian_weight():
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in PILOT_PATCH_SIZE
    ]
    grids = np.meshgrid(*coordinates, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    weight = np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32)
    return np.maximum(weight, 0.05)


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


@torch.no_grad()
def pilot_full_volume_probability(study_model, row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: validation channel order changed.')
    region = pilot_load_region_map(subject_id)
    spatial_shape = image.shape[1:]
    probability_sum = np.zeros(spatial_shape, dtype=np.float32)
    weight_sum = np.zeros(spatial_shape, dtype=np.float32)
    window_records = []
    for x in pilot_sliding_starts(spatial_shape[0], PILOT_PATCH_SIZE[0]):
        for y in pilot_sliding_starts(spatial_shape[1], PILOT_PATCH_SIZE[1]):
            for z in pilot_sliding_starts(spatial_shape[2], PILOT_PATCH_SIZE[2]):
                window_records.append((x, y, z))
    study_model.eval()
    for start_index in range(0, len(window_records), 2):
        chunk = window_records[start_index:start_index + 2]
        numpy_inputs = []
        mappings = []
        for starts in chunk:
            image_patch, source_slices, destination_slices = pilot_extract_window(
                image, starts, (1, 2, 3)
            )
            region_patch, _, _ = pilot_extract_window(region, starts, (0, 1, 2))
            cow_patch, _, _ = pilot_extract_window(cow, starts, (0, 1, 2))
            lvo_patch, _, _ = pilot_extract_window(lvo, starts, (0, 1, 2))
            numpy_inputs.append(pilot_model_input_from_patches(
                row, image_patch, region_patch, cow_patch, lvo_patch,
                lvo_present, apply_graph_mask=False,
            ))
            mappings.append((source_slices, destination_slices))
        model_batch = {
            key: torch.from_numpy(np.stack([
                np.asarray(item[key]) for item in numpy_inputs
            ])).to(device, non_blocking=True)
            for key in MODEL_INPUT_KEYS
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            output = study_model(
                **model_batch,
                force_atlas_dropout=False,
                force_vascular_dropout=False,
                force_occlusion_dropout=False,
                force_clinical_dropout=False,
            )
        probabilities = torch.sigmoid(output['logits'].float())[
            :, 0
        ].cpu().numpy()
        for local_index, probability_patch in enumerate(probabilities):
            source_slices, destination_slices = mappings[local_index]
            patch_probability = probability_patch[destination_slices]
            patch_weight = PILOT_WINDOW_WEIGHT[destination_slices]
            probability_sum[source_slices] += patch_probability * patch_weight
            weight_sum[source_slices] += patch_weight
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    if not np.isfinite(probability).all() or bool((weight_sum <= 0).any()):
        raise RuntimeError(f'{subject_id}: invalid sliding-window output.')
    return probability.astype(np.float32), target_binary, spacing


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap_pairs = np.column_stack([
        predicted_labels[target & prediction],
        target_labels[target & prediction],
    ])
    overlap_pairs = overlap_pairs[
        (overlap_pairs[:, 0] > 0) & (overlap_pairs[:, 1] > 0)
    ]
    if len(overlap_pairs) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap_pairs, axis=0, return_counts=True)
    order = np.argsort(-counts)
    used_predicted = set()
    used_target = set()
    matches = 0
    for pair_index in order:
        predicted_id, target_id = [int(value) for value in pairs[pair_index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


PILOT_THRESHOLDS = np.round(np.arange(0.10, 0.81, 0.05), 2)


@torch.no_grad()
def pilot_validate(study_model, epoch):
    case_cache = []
    for case_index, row in enumerate(heldout_rows, start=1):
        probability, target, spacing = pilot_full_volume_probability(
            study_model, row
        )
        case_cache.append({
            'row': row,
            'probability': probability.astype(np.float16),
            'target': target,
            'spacing': spacing,
        })
        print(
            f'  Held-out inference {case_index:02d}/{len(heldout_rows)}: '
            f'{row["subject_id"]}'
        )
    threshold_rows = []
    case_threshold_rows = []
    for threshold in PILOT_THRESHOLDS:
        positive_dice = []
        global_intersection = 0
        global_prediction = 0
        global_target = 0
        for case in case_cache:
            row = case['row']
            prediction = case['probability'] >= threshold
            target = case['target']
            intersection = int((prediction & target).sum())
            predicted_voxels = int(prediction.sum())
            target_voxels = int(target.sum())
            dice = (
                (2.0 * intersection) / (predicted_voxels + target_voxels)
                if predicted_voxels + target_voxels > 0
                else 1.0
            )
            hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
            if hard_metric_eligible and target_voxels > 0:
                positive_dice.append(dice)
                global_intersection += intersection
                global_prediction += predicted_voxels
                global_target += target_voxels
            case_threshold_rows.append({
                'epoch': epoch,
                'subject_id': str(row['subject_id']),
                'threshold': float(threshold),
                'hard_metric_eligible': hard_metric_eligible,
                'target_voxels': target_voxels,
                'predicted_voxels': predicted_voxels,
                'intersection_voxels': intersection,
                'dice': float(dice),
            })
        if not positive_dice:
            raise RuntimeError('Fold-0 held-out set has no positive hard targets.')
        global_dice = (
            2.0 * global_intersection
            / max(global_prediction + global_target, 1)
        )
        threshold_rows.append({
            'threshold': float(threshold),
            'positive_mean_dice': float(np.mean(positive_dice)),
            'positive_median_dice': float(np.median(positive_dice)),
            'positive_global_dice': float(global_dice),
            'positive_subjects': len(positive_dice),
        })
    threshold_df = pd.DataFrame(threshold_rows).sort_values(
        ['positive_mean_dice', 'positive_global_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected_threshold = float(threshold_df.iloc[0]['threshold'])
    fixed_threshold_row = threshold_df.loc[
        np.isclose(threshold_df['threshold'], 0.50)
    ].iloc[0]
    selected_case_rows = []
    lesion_tp = lesion_fp = lesion_fn = 0
    total_intersection = total_prediction = total_target = 0
    positive_dice = []
    precisions = []
    recalls = []
    volume_errors = []
    empty_fp_volumes = []
    bin_dice = {name: [] for name in POSITIVE_BINS}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        prediction = probability >= selected_threshold
        target = case['target']
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0
            else 1.0
        )
        precision = intersection / max(predicted_voxels, 1)
        recall = intersection / max(target_voxels, 1)
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        predicted_volume_ml = predicted_voxels * voxel_volume_ml
        target_volume_ml = target_voxels * voxel_volume_ml
        hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
        true_positive_lesions = false_positive_lesions = false_negative_lesions = 0
        if hard_metric_eligible:
            (
                true_positive_lesions,
                false_positive_lesions,
                false_negative_lesions,
            ) = pilot_lesion_counts(prediction, target)
            lesion_tp += true_positive_lesions
            lesion_fp += false_positive_lesions
            lesion_fn += false_negative_lesions
            volume_errors.append(abs(predicted_volume_ml - target_volume_ml))
            if target_voxels > 0:
                positive_dice.append(dice)
                precisions.append(precision)
                recalls.append(recall)
                total_intersection += intersection
                total_prediction += predicted_voxels
                total_target += target_voxels
                lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
                if lesion_bin in bin_dice:
                    bin_dice[lesion_bin].append(dice)
            else:
                empty_fp_volumes.append(predicted_volume_ml)
        selected_case_rows.append({
            'epoch': epoch,
            'subject_id': str(row['subject_id']),
            'threshold': selected_threshold,
            'supervision_mode': str(row['trial8_supervision_mode']),
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': hard_metric_eligible,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'precision': float(precision),
            'recall': float(recall),
            'target_volume_ml': target_volume_ml,
            'predicted_volume_ml': predicted_volume_ml,
            'absolute_volume_error_ml': abs(
                predicted_volume_ml - target_volume_ml
            ),
            'lesion_true_positive': true_positive_lesions,
            'lesion_false_positive': false_positive_lesions,
            'lesion_false_negative': false_negative_lesions,
        })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    result = {
        'threshold': selected_threshold,
        'positive_mean_dice': float(np.mean(positive_dice)),
        'positive_median_dice': float(np.median(positive_dice)),
        'positive_global_dice': float(
            2.0 * total_intersection / max(total_prediction + total_target, 1)
        ),
        'fixed_0p50_positive_mean_dice': float(
            fixed_threshold_row['positive_mean_dice']
        ),
        'fixed_0p50_positive_global_dice': float(
            fixed_threshold_row['positive_global_dice']
        ),
        'calibration_mean_dice_gain': float(
            np.mean(positive_dice)
            - float(fixed_threshold_row['positive_mean_dice'])
        ),
        'mean_voxel_precision': float(np.mean(precisions)),
        'mean_voxel_recall': float(np.mean(recalls)),
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'mean_absolute_volume_error_ml': float(np.mean(volume_errors)),
        'mean_empty_false_positive_volume_ml': float(
            np.mean(empty_fp_volumes) if empty_fp_volumes else 0.0
        ),
        'positive_subjects': len(positive_dice),
        'hard_metric_subjects': int(sum(
            as_bool(case['row']['trial8_hard_metric_eligible'])
            for case in case_cache
        )),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(threshold_df, epoch_root / 'threshold_summary.csv')
    atomic_write_csv(
        pd.DataFrame(case_threshold_rows), epoch_root / 'case_threshold_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(selected_case_rows), epoch_root / 'selected_case_metrics.csv'
    )
    atomic_write_json(result, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result






MAX_EPOCHS = int(fivefold_policy['maximum_epochs'])
PATIENCE_VALIDATIONS = int(
    fivefold_policy['early_stopping_patience_validations']
)
EARLY_STOPPING_MINIMUM_EPOCH = int(
    fivefold_policy['early_stopping_minimum_epoch']
)
EARLY_STOPPING_MINIMUM_DELTA = float(
    fivefold_policy['early_stopping_minimum_delta']
)
BASE_LEARNING_RATE = float(fivefold_policy['learning_rate'])
WEIGHT_DECAY = float(fivefold_policy['weight_decay'])
ACCUMULATION_STEPS = int(fivefold_policy['gradient_accumulation_steps'])
GRADIENT_CLIP_NORM = float(fivefold_policy['gradient_clip_norm'])
EMA_DECAY = float(fivefold_policy['ema_decay'])
BATCH_SIZE = int(fivefold_policy['batch_size'])
GATE_LEARNING_RATE_MULTIPLIER = 5.0
NUM_WORKERS = 0

pilot_policy_checks = {
    'patch_size_matches': tuple(fivefold_policy['patch_size']) == PILOT_PATCH_SIZE,
    'batch_size_is_one': BATCH_SIZE == 1,
    'accumulation_steps_are_four': ACCUMULATION_STEPS == 4,
    'maximum_epochs_are_60': MAX_EPOCHS == 60,
    'validation_every_epoch': int(
        fivefold_policy['validation_interval_epochs']
    ) == 1,
    'metrics_printed_every_epoch': fivefold_policy[
        'metrics_printed_every_epoch'
    ] is True,
    'early_stopping_minimum_epoch_is_20': (
        EARLY_STOPPING_MINIMUM_EPOCH == 20
    ),
    'early_stopping_patience_is_15': PATIENCE_VALIDATIONS == 15,
    'early_stopping_delta_is_0p002': math.isclose(
        EARLY_STOPPING_MINIMUM_DELTA, 0.002
    ),
    'volume_case_weights_disabled': fivefold_policy[
        'volume_case_loss_weighting'
    ] is False,
    'top_k_ohem_disabled': fivefold_policy['loss']['top_k_ohem'] is False,
    'soft_dice_weight_is_0p50': math.isclose(
        float(fivefold_policy['loss']['soft_dice_weight']), 0.50
    ),
    'soft_focal_bce_weight_is_0p30': math.isclose(
        float(fivefold_policy['loss']['soft_focal_bce_weight']), 0.30
    ),
    'hard_focal_tversky_weight_is_0p20': math.isclose(
        float(fivefold_policy['loss']['hard_focal_tversky_weight']), 0.20
    ),
    'focal_tversky_parameters_match': (
        math.isclose(
            float(fivefold_policy['loss'][
                'hard_focal_tversky_alpha_false_positive'
            ]),
            0.30,
        )
        and math.isclose(
            float(fivefold_policy['loss'][
                'hard_focal_tversky_beta_false_negative'
            ]),
            0.70,
        )
        and math.isclose(
            float(fivefold_policy['loss']['hard_focal_tversky_gamma']),
            0.75,
        )
    ),
    'deep_supervision_weights_match': (
        math.isclose(
            float(fivefold_policy['deep_supervision_weights'][
                'half_resolution'
            ]),
            0.15,
        )
        and math.isclose(
            float(fivefold_policy['deep_supervision_weights'][
                'quarter_resolution'
            ]),
            0.10,
        )
    ),
    'territory_node_loss_policy_matches': (
        math.isclose(float(fivefold_policy['territory_node_loss_weight']), 0.02)
        and math.isclose(float(fivefold_policy['territory_positive_weight']), 4.0)
    ),
    'graph_feature_mask_probability_is_0p05': math.isclose(
        PILOT_GRAPH_FEATURE_MASK_PROBABILITY, 0.05
    ),
    'atlas_dropout_probability_is_0p25': math.isclose(
        PILOT_ATLAS_DROPOUT_PROBABILITY, 0.25
    ),
    'spatial_augmentation_policy_matches': (
        math.isclose(float(fivefold_policy['spatial_affine_probability']), 0.60)
        and math.isclose(
            float(fivefold_policy['spatial_rotation_degrees']), 8.0
        )
        and list(fivefold_policy['spatial_scale_range']) == [0.92, 1.08]
        and math.isclose(
            float(fivefold_policy['spatial_translation_fraction_xy']), 0.05
        )
        and math.isclose(
            float(fivefold_policy['spatial_translation_fraction_z']), 0.025
        )
        and fivefold_policy[
            'post_augmentation_positive_support_required'
        ] is True
        and fivefold_policy[
            'augmentation_retry_then_identity_fallback'
        ] is True
    ),
    'left_right_reflection_disabled': fivefold_policy[
        'left_right_reflection'
    ] is False,
    'checkpoint_metric_matches': fivefold_policy[
        'checkpoint_metric'
    ] == 'heldout_positive_mean_dice',
    'locked_validation_policy_preserved': 'do not read' in fivefold_policy[
        'locked_validation_policy'
    ],
    'test_policy_preserved': 'sealed' in fivefold_policy['test_policy'],
}
if not all(pilot_policy_checks.values()):
    failed = [name for name, passed in pilot_policy_checks.items() if not passed]
    raise RuntimeError('Frozen pilot policy changed: ' + ', '.join(failed))

set_determinism(PILOT_SEED)
training_dataset = Trial8Fold0PatchDataset()
epoch1_schedule = training_dataset.schedule
schedule_rows = [{
    'sample_index': index,
    'sample_type': entry['sample_type'],
    'subject_id': entry['subject_id'],
    'center_x': int(entry['center'][0]),
    'center_y': int(entry['center'][1]),
    'center_z': int(entry['center'][2]),
    'topology_relation': entry['topology_relation'],
} for index, entry in enumerate(epoch1_schedule)]
atomic_write_csv(pd.DataFrame(schedule_rows), PILOT_SCHEDULE_AUDIT_PATH)

pilot_schedule_checks = {
    'epoch_has_168_patches': len(epoch1_schedule) == 168,
    'positive_patches_are_112': training_dataset.counts['positive'] == 112,
    'topology_negative_patches_are_28': (
        training_dataset.counts['topology_negative'] == 28
    ),
    'brain_negative_patches_are_28': (
        training_dataset.counts['brain_negative'] == 28
    ),
    'positive_fraction_is_two_thirds': math.isclose(
        training_dataset.counts['positive'] / len(epoch1_schedule),
        2.0 / 3.0,
    ),
    'all_fit_positive_subjects_covered': set(
        fold0_fit_df.loc[
            fold0_fit_df['trial8_sampling_positive'].map(as_bool), 'subject_id'
        ].astype(str)
    ).issubset({
        entry['subject_id'] for entry in epoch1_schedule
        if entry['sample_type'] == 'positive'
    }),
}
if not all(pilot_schedule_checks.values()):
    failed = [name for name, passed in pilot_schedule_checks.items() if not passed]
    raise RuntimeError('Fold-0 epoch schedule changed: ' + ', '.join(failed))

training_model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=15,
    territory_feature_dim=65,
    vessel_feature_dim=15,
    case_feature_dim=8,
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=PILOT_ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)
if parameter_count(training_model) != EXPECTED_V2_PARAMETERS:
    raise RuntimeError('Fold-0 pilot model parameter count changed.')

expected_high_lr_gate_parameter_names = {
    f'{prefix}.gate.network.2.{suffix}'
    for prefix in [
        'vascular_fusion_quarter',
        'vascular_fusion_eighth',
        'vascular_fusion_bottleneck',
        'context_fusion',
    ]
    for suffix in ['weight', 'bias']
}
observed_high_lr_gate_parameter_names = {
    name for name, _ in training_model.named_parameters()
    if '.gate.network.2.' in name
}
pilot_model_training_contract_checks = {
    'training_atlas_dropout_matches_frozen_policy': math.isclose(
        training_model.atlas_dropout,
        PILOT_ATLAS_DROPOUT_PROBABILITY,
    ),
    'new_branch_dropout_probabilities_match_v2_contract': (
        math.isclose(training_model.vascular_dropout, 0.15)
        and math.isclose(training_model.occlusion_dropout, 0.20)
        and math.isclose(training_model.clinical_dropout, 0.20)
    ),
    'high_lr_group_is_exactly_four_new_fusion_gates': (
        observed_high_lr_gate_parameter_names
        == expected_high_lr_gate_parameter_names
    ),
    'all_initialized_parameters_are_trainable': all(
        parameter.requires_grad for parameter in training_model.parameters()
    ),
}

smoke_loader = torch.utils.data.DataLoader(
    training_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == 'cuda',
)
smoke_cpu_batch = next(iter(smoke_loader))
training_model.train()
with torch.autocast(
    device_type=device.type,
    dtype=torch.bfloat16,
    enabled=device.type == 'cuda',
):
    pilot_smoke_loss, pilot_smoke_parts = pilot_compute_training_loss(
        training_model, smoke_cpu_batch
    )
if not bool(torch.isfinite(pilot_smoke_loss).item()):
    raise RuntimeError('Fold-0 pilot smoke loss is non-finite.')
pilot_smoke_loss.backward()
pilot_smoke_gradient_finite = all(
    parameter.grad is None or bool(torch.isfinite(parameter.grad).all().item())
    for parameter in training_model.parameters()
)
pilot_smoke_gradient_nonzero = any(
    parameter.grad is not None and bool((parameter.grad != 0).any().item())
    for parameter in training_model.parameters()
)
training_model.zero_grad(set_to_none=True)

pilot_start_checks = {
    **pilot_upstream_checks,
    **pilot_manifest_checks,
    **pilot_scaler_contract_checks,
    **pilot_target_checks,
    **pilot_policy_checks,
    **pilot_schedule_checks,
    **pilot_model_training_contract_checks,
    'model_parameter_count_matches': parameter_count(training_model)
    == EXPECTED_V2_PARAMETERS,
    'smoke_loss_finite': bool(torch.isfinite(pilot_smoke_loss).item()),
    'smoke_gradients_finite': pilot_smoke_gradient_finite,
    'smoke_has_nonzero_gradient': pilot_smoke_gradient_nonzero,
    'target_access_is_development_train_only': True,
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
    'outcome_workbooks_not_read': True,
    'expert_routing_disabled': True,
    'direct_graph_to_logit_residual_disabled': True,
}
pilot_training_authorized = bool(all(pilot_start_checks.values()))
atomic_write_json({
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'pilot_version': PILOT_VERSION,
    'code_revision': PILOT_CODE_REVISION,
    'pilot_training_authorized': pilot_training_authorized,
    'checks': pilot_start_checks,
    'fold': HELDOUT_FOLD,
    'isolated_construction_output_root': str(OUTPUT_ROOT),
    'isolated_pilot_output_root': str(PILOT_ROOT),
    'legacy_v2_pilot_root': str(LEGACY_V2_PILOT_ROOT),
    'fit_subjects': len(fold0_fit_df),
    'heldout_subjects': len(fold0_heldout_df),
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'outcome_workbooks_read': False,
    'full_fivefold_training_authorized': False,
}, PILOT_START_GATE_PATH)
if not pilot_training_authorized:
    failed = [name for name, passed in pilot_start_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2.1 FOLD-0 PILOT START GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ema_model = copy.deepcopy(training_model).to(device).eval()
ema_model.requires_grad_(False)
gate_parameters = []
standard_parameters = []
for name, parameter in training_model.named_parameters():
    if name in expected_high_lr_gate_parameter_names:
        gate_parameters.append(parameter)
    else:
        standard_parameters.append(parameter)
if not gate_parameters:
    raise RuntimeError('Zero-initialized fusion-gate parameters were not found.')
training_optimizer = torch.optim.AdamW([
    {
        'params': standard_parameters,
        'lr': BASE_LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'group_name': 'standard',
    },
    {
        'params': gate_parameters,
        'lr': BASE_LEARNING_RATE * GATE_LEARNING_RATE_MULTIPLIER,
        'weight_decay': 0.0,
        'group_name': 'zero_init_fusion_gates',
    },
])
updates_per_epoch = math.ceil(len(training_dataset) / ACCUMULATION_STEPS)
total_updates = MAX_EPOCHS * updates_per_epoch
warmup_updates = 3 * updates_per_epoch


def pilot_lr_lambda(update_index):
    completed = update_index + 1
    if completed <= warmup_updates:
        return completed / max(warmup_updates, 1)
    progress = (completed - warmup_updates) / max(
        total_updates - warmup_updates, 1
    )
    return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


training_scheduler = torch.optim.lr_scheduler.LambdaLR(
    training_optimizer, lr_lambda=pilot_lr_lambda
)


def pilot_update_ema(ema, study, maximum_decay, update_step):
    decay = min(maximum_decay, (1.0 + update_step) / (10.0 + update_step))
    with torch.no_grad():
        for ema_parameter, parameter in zip(
            ema.parameters(), study.parameters()
        ):
            ema_parameter.mul_(decay).add_(parameter, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(ema.buffers(), study.buffers()):
            ema_buffer.copy_(buffer)


def pilot_epoch_loader(epoch):
    training_dataset.set_epoch(epoch)
    generator = torch.Generator()
    generator.manual_seed(PILOT_SEED + epoch * 97_409)
    return torch.utils.data.DataLoader(
        training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda',
        persistent_workers=False,
        generator=generator,
    )


pilot_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'pilot_version': PILOT_VERSION,
    'code_revision': PILOT_CODE_REVISION,
    'fold': HELDOUT_FOLD,
    'isolated_construction_output_root': str(OUTPUT_ROOT),
    'isolated_pilot_output_root': str(PILOT_ROOT),
    'legacy_v2_pilot_root': str(LEGACY_V2_PILOT_ROOT),
    'fit_subject_count': len(fold0_fit_df),
    'heldout_subject_count': len(fold0_heldout_df),
    'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
    'heldout_subject_hash': pilot_subject_hash(fold0_heldout_df['subject_id']),
    'multimodal_manifest_sha256': multimodal_manifest_sha256,
    'fold_manifest_sha256': fold_manifest_sha256,
    'loader_gate_sha256': loader_gate_sha256,
    'fivefold_gate_sha256': fivefold_gate_sha256,
    'v2_model_gate_sha256': v2_model_gate_sha256,
    'architecture_sha256': architecture_sha256,
    'architecture_contract_sha256': architecture_contract_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'fold0_graph_scaler_sha256': fold0_graph_scaler_sha256,
    'fold0_clinical_scaler_sha256': fold0_clinical_scaler_sha256,
    'parameters': EXPECTED_V2_PARAMETERS,
    'patch_size': list(PILOT_PATCH_SIZE),
    'samples_per_epoch': len(training_dataset),
    'batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': ACCUMULATION_STEPS,
    'maximum_epochs': MAX_EPOCHS,
    'base_learning_rate': BASE_LEARNING_RATE,
    'gate_learning_rate_multiplier': GATE_LEARNING_RATE_MULTIPLIER,
    'training_atlas_dropout_probability': (
        PILOT_ATLAS_DROPOUT_PROBABILITY
    ),
    'graph_feature_mask_probability': PILOT_GRAPH_FEATURE_MASK_PROBABILITY,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_decay': EMA_DECAY,
    'early_stopping_minimum_epoch': EARLY_STOPPING_MINIMUM_EPOCH,
    'early_stopping_patience': PATIENCE_VALIDATIONS,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'checkpoint_metric': 'heldout_positive_mean_dice',
    'threshold_grid': PILOT_THRESHOLDS.tolist(),
    'mixed_precision': 'bfloat16_on_cuda_float32_reductions',
    'spatial_augmentation_maximum_attempts': 4,
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'full_fivefold_training_authorized': False,
}
if PILOT_CONFIG_PATH.exists():
    existing_config = json.loads(PILOT_CONFIG_PATH.read_text(encoding='utf-8'))
    immutable_keys = [
        'pilot_version', 'code_revision', 'fold',
        'isolated_construction_output_root', 'isolated_pilot_output_root',
        'fit_subject_hash', 'heldout_subject_hash',
        'multimodal_manifest_sha256', 'fold_manifest_sha256',
        'loader_gate_sha256', 'fivefold_gate_sha256',
        'architecture_contract_sha256', 'graph_schema_sha256',
        'fold0_graph_scaler_sha256', 'fold0_clinical_scaler_sha256',
        'parameters', 'patch_size', 'samples_per_epoch', 'batch_size',
        'gradient_accumulation_steps', 'maximum_epochs',
        'base_learning_rate', 'gate_learning_rate_multiplier',
        'training_atlas_dropout_probability',
        'graph_feature_mask_probability',
        'weight_decay', 'gradient_clip_norm', 'ema_decay',
        'early_stopping_minimum_epoch', 'early_stopping_patience',
        'early_stopping_minimum_delta', 'checkpoint_metric',
        'threshold_grid', 'mixed_precision',
        'spatial_augmentation_maximum_attempts',
    ]
    if any(
        existing_config.get(key) != pilot_config.get(key)
        for key in immutable_keys
    ):
        raise RuntimeError('Existing fold-0 pilot configuration changed.')
else:
    atomic_write_json(pilot_config, PILOT_CONFIG_PATH)

start_epoch = 1
global_update_step = 0
best_mean_dice = -math.inf
best_epoch = 0
best_threshold = 0.50
validations_without_improvement = 0
history_rows = []
if PILOT_LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    checkpoint_checks = {
        'pilot_version_matches': checkpoint.get('pilot_version') == PILOT_VERSION,
        'code_revision_matches': checkpoint.get('code_revision')
        == PILOT_CODE_REVISION,
        'manifest_hash_matches': checkpoint.get('multimodal_manifest_sha256')
        == multimodal_manifest_sha256,
        'fold_manifest_hash_matches': checkpoint.get(
            'fold_manifest_sha256'
        ) == fold_manifest_sha256,
        'loader_gate_hash_matches': checkpoint.get('loader_gate_sha256')
        == loader_gate_sha256,
        'fivefold_gate_hash_matches': checkpoint.get(
            'fivefold_gate_sha256'
        ) == fivefold_gate_sha256,
        'graph_schema_hash_matches': checkpoint.get(
            'graph_schema_sha256'
        ) == safe_schema_sha256,
        'graph_scaler_hash_matches': checkpoint.get(
            'fold0_graph_scaler_sha256'
        ) == fold0_graph_scaler_sha256,
        'clinical_scaler_hash_matches': checkpoint.get(
            'fold0_clinical_scaler_sha256'
        ) == fold0_clinical_scaler_sha256,
        'architecture_contract_hash_matches': checkpoint.get(
            'architecture_contract_sha256'
        ) == architecture_contract_sha256,
        'parameter_count_matches': checkpoint.get('parameters')
        == EXPECTED_V2_PARAMETERS,
        'locked_validation_targets_not_read': checkpoint.get(
            'locked_validation_targets_read'
        ) is False,
        'test_targets_not_read': checkpoint.get('test_targets_read') is False,
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Fold-0 checkpoint mismatch: ' + ', '.join(failed))
    training_model.load_state_dict(checkpoint['model_state_dict'])
    ema_model.load_state_dict(checkpoint['ema_model_state_dict'])
    training_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    training_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    pilot_restore_rng_state(checkpoint['rng_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    global_update_step = int(checkpoint['global_update_step'])
    best_mean_dice = float(checkpoint['best_mean_dice'])
    best_epoch = int(checkpoint['best_epoch'])
    best_threshold = float(checkpoint['best_threshold'])
    validations_without_improvement = int(
        checkpoint['validations_without_improvement']
    )
    history_rows = list(checkpoint.get('history_rows', []))
    print(
        f'Resuming fold-0 pilot after epoch {start_epoch - 1}; '
        f'best EMA Dice={best_mean_dice:.4f} at epoch {best_epoch}.'
    )
else:
    print('Starting fold-0 pilot from a fresh deterministic initialization.')






print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2.1 FOLD-0 PILOT TRAINING START')
print('=' * 80)
print(f'Device / GPU:                       {device} / ' + (
    torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'
))
print(f'Fit / held-out / locked val / test: {len(fold0_fit_df)} / {len(fold0_heldout_df)} / 0 / 0 targets')
print(f'Patch size:                         {PILOT_PATCH_SIZE}')
print(f'Samples per epoch:                  {len(training_dataset)} (112/28/28)')
print(f'Batch / accumulation / effective:   {BATCH_SIZE} / {ACCUMULATION_STEPS} / {BATCH_SIZE * ACCUMULATION_STEPS}')
print(f'Maximum epochs:                     {MAX_EPOCHS}')
print(f'Early stopping:                     epoch >= {EARLY_STOPPING_MINIMUM_EPOCH}, patience={PATIENCE_VALIDATIONS}, delta={EARLY_STOPPING_MINIMUM_DELTA:.3f}')
print(f'Base/gate learning rates:           {BASE_LEARNING_RATE:.1e} / {BASE_LEARNING_RATE * GATE_LEARNING_RATE_MULTIPLIER:.1e}')
print(f'Training atlas dropout:             {PILOT_ATLAS_DROPOUT_PROBABILITY:.2f}')
print(f'Total parameters:                   {EXPECTED_V2_PARAMETERS:,}')
print('Validation:                         full-volume EMA every epoch')
print('Threshold calibration:              held-out fold only, 0.10..0.80')
print('Locked validation/test targets:     sealed')

training_already_complete = (
    start_epoch > MAX_EPOCHS
    or (
        start_epoch - 1 >= EARLY_STOPPING_MINIMUM_EPOCH
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    )
)
stop_reason = (
    'already_complete_on_resume'
    if training_already_complete
    else 'maximum_epochs_reached'
)

for epoch in range(
    start_epoch,
    start_epoch if training_already_complete else MAX_EPOCHS + 1,
):
    epoch_start = datetime.now(timezone.utc)
    loader = pilot_epoch_loader(epoch)
    expected_counts = dict(training_dataset.counts)
    training_model.train()
    training_optimizer.zero_grad(set_to_none=True)
    running_names = [
        'total_loss', 'main_loss', 'soft_dice_loss',
        'soft_focal_bce_loss', 'hard_focal_tversky_loss',
        'hard_contribution', 'half_loss', 'quarter_loss', 'node_loss',
        'atlas_retained_fraction', 'vascular_retained_fraction',
        'occlusion_retained_fraction', 'clinical_retained_fraction',
        'context_gate_absolute_mean', 'quarter_gate_absolute_mean',
        'eighth_gate_absolute_mean', 'bottleneck_gate_absolute_mean',
        'spatially_augmented_fraction',
    ]
    running = {name: 0.0 for name in running_names}
    processed_batches = 0
    processed_samples = 0
    observed_counts = {
        'positive': 0, 'topology_negative': 0, 'brain_negative': 0,
    }
    gradient_norms = []
    accumulation_subjects = []
    for batch_index, cpu_batch in enumerate(loader, start=1):
        processed_samples += int(cpu_batch['target_binary'].shape[0])
        for sample_type in cpu_batch['sample_type']:
            observed_counts[str(sample_type)] += 1
        accumulation_subjects.extend(str(value) for value in cpu_batch['subject_id'])
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            total_loss, parts = pilot_compute_training_loss(
                training_model, cpu_batch
            )
            group_start = ((batch_index - 1) // ACCUMULATION_STEPS) * (
                ACCUMULATION_STEPS
            )
            group_size = min(
                ACCUMULATION_STEPS, len(loader) - group_start
            )
            scaled_loss = total_loss / group_size
        scaled_loss.backward()
        for name in running_names:
            running[name] += float(parts[name].detach().cpu())
        processed_batches += 1
        should_update = (
            batch_index % ACCUMULATION_STEPS == 0
            or batch_index == len(loader)
        )
        if should_update:
            gradient_norm = float(torch.nn.utils.clip_grad_norm_(
                training_model.parameters(), GRADIENT_CLIP_NORM
            ).item())
            if not math.isfinite(gradient_norm):
                training_optimizer.zero_grad(set_to_none=True)
                raise RuntimeError(
                    f'Non-finite gradient at epoch {epoch}, batch '
                    f'{batch_index}, subjects={accumulation_subjects}.'
                )
            gradient_norms.append(gradient_norm)
            training_optimizer.step()
            training_optimizer.zero_grad(set_to_none=True)
            training_scheduler.step()
            global_update_step += 1
            pilot_update_ema(
                ema_model, training_model, EMA_DECAY, global_update_step
            )
            accumulation_subjects.clear()
    if observed_counts != expected_counts or processed_samples != len(training_dataset):
        raise RuntimeError(
            f'Epoch {epoch}: sampling counts changed. '
            f'Observed {observed_counts}; expected {expected_counts}.'
        )
    epoch_means = {
        name: running[name] / max(processed_batches, 1)
        for name in running_names
    }
    if not gradient_norms or not all(
        math.isfinite(value) for value in epoch_means.values()
    ):
        raise RuntimeError(f'Epoch {epoch}: non-finite training state.')

    print(f'Running full fold-0 EMA validation at epoch {epoch}...')
    validation_result = pilot_validate(ema_model, epoch)
    current_dice = float(validation_result['positive_mean_dice'])
    current_threshold = float(validation_result['threshold'])
    improved = current_dice > best_mean_dice + EARLY_STOPPING_MINIMUM_DELTA
    if improved:
        best_mean_dice = current_dice
        best_epoch = epoch
        best_threshold = current_threshold
        validations_without_improvement = 0
    else:
        validations_without_improvement += 1
    epoch_seconds = (
        datetime.now(timezone.utc) - epoch_start
    ).total_seconds()
    history_row = {
        'epoch': epoch,
        'global_update_step': global_update_step,
        'standard_learning_rate': float(training_optimizer.param_groups[0]['lr']),
        'fusion_gate_learning_rate': float(training_optimizer.param_groups[1]['lr']),
        **epoch_means,
        'mean_gradient_norm': float(np.mean(gradient_norms)),
        'maximum_gradient_norm': float(np.max(gradient_norms)),
        'positive_patch_fraction': observed_counts['positive'] / processed_samples,
        'topology_negative_fraction': (
            observed_counts['topology_negative'] / processed_samples
        ),
        'brain_negative_fraction': (
            observed_counts['brain_negative'] / processed_samples
        ),
        'validation_threshold': current_threshold,
        'validation_positive_mean_dice': current_dice,
        'validation_positive_median_dice': validation_result[
            'positive_median_dice'
        ],
        'validation_positive_global_dice': validation_result[
            'positive_global_dice'
        ],
        'validation_fixed_0p50_positive_mean_dice': validation_result[
            'fixed_0p50_positive_mean_dice'
        ],
        'validation_fixed_0p50_positive_global_dice': validation_result[
            'fixed_0p50_positive_global_dice'
        ],
        'validation_calibration_mean_dice_gain': validation_result[
            'calibration_mean_dice_gain'
        ],
        'validation_mean_voxel_precision': validation_result[
            'mean_voxel_precision'
        ],
        'validation_mean_voxel_recall': validation_result[
            'mean_voxel_recall'
        ],
        'validation_lesion_precision': validation_result['lesion_precision'],
        'validation_lesion_recall': validation_result['lesion_recall'],
        'validation_lesion_f1': validation_result['lesion_f1'],
        'validation_mean_absolute_volume_error_ml': validation_result[
            'mean_absolute_volume_error_ml'
        ],
        'validation_mean_empty_false_positive_volume_ml': validation_result[
            'mean_empty_false_positive_volume_ml'
        ],
        'best_positive_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'improved': improved,
        'epoch_seconds': epoch_seconds,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    for lesion_bin, value in validation_result['lesion_bin_mean_dice'].items():
        history_row[f'validation_mean_dice_{lesion_bin}'] = value
    history_rows.append(to_builtin(history_row))
    atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)

    checkpoint_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'epoch': epoch,
        'global_update_step': global_update_step,
        'model_state_dict': training_model.state_dict(),
        'ema_model_state_dict': ema_model.state_dict(),
        'optimizer_state_dict': training_optimizer.state_dict(),
        'scheduler_state_dict': training_scheduler.state_dict(),
        'rng_state': pilot_capture_rng_state(),
        'best_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'history_rows': history_rows,
        'multimodal_manifest_sha256': multimodal_manifest_sha256,
        'fold_manifest_sha256': fold_manifest_sha256,
        'loader_gate_sha256': loader_gate_sha256,
        'fivefold_gate_sha256': fivefold_gate_sha256,
        'architecture_sha256': architecture_sha256,
        'architecture_contract_sha256': architecture_contract_sha256,
        'graph_schema_sha256': safe_schema_sha256,
        'fold0_graph_scaler_sha256': fold0_graph_scaler_sha256,
        'fold0_clinical_scaler_sha256': fold0_clinical_scaler_sha256,
        'parameters': EXPECTED_V2_PARAMETERS,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
    }
    pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
    if improved:
        pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)

    print(
        f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
        f'loss={epoch_means["total_loss"]:.4f}, '
        f'main={epoch_means["main_loss"]:.4f}, '
        f'node={epoch_means["node_loss"]:.4f}, '
        f'lr={history_row["standard_learning_rate"]:.2e}, '
        f'gate-lr={history_row["fusion_gate_learning_rate"]:.2e}, '
        f'grad={history_row["mean_gradient_norm"]:.3f}, '
        f'time={epoch_seconds / 60.0:.1f} min'
    )
    print(
        f'  EMA held-out: mean Dice={current_dice:.4f}, '
        f'median Dice={validation_result["positive_median_dice"]:.4f}, '
        f'global Dice={validation_result["positive_global_dice"]:.4f}, '
        f'voxel P/R={validation_result["mean_voxel_precision"]:.4f}/'
        f'{validation_result["mean_voxel_recall"]:.4f}, '
        f'lesion F1={validation_result["lesion_f1"]:.4f}, '
        f'threshold={current_threshold:.2f}'
    )
    print(
        '  Fixed 0.50 mean/global Dice: '
        f'{validation_result["fixed_0p50_positive_mean_dice"]:.4f}/'
        f'{validation_result["fixed_0p50_positive_global_dice"]:.4f}; '
        'calibration gain='
        f'{validation_result["calibration_mean_dice_gain"]:+.4f}'
    )
    print(
        f'  Volume MAE={validation_result["mean_absolute_volume_error_ml"]:.3f} mL, '
        f'empty FP={validation_result["mean_empty_false_positive_volume_ml"]:.3f} mL, '
        f'best={best_mean_dice:.4f} at epoch {best_epoch}, '
        f'patience={validations_without_improvement}/{PATIENCE_VALIDATIONS}'
    )
    print(
        '  Retained atlas/vascular/occlusion/clinical: '
        f'{epoch_means["atlas_retained_fraction"]:.3f}/'
        f'{epoch_means["vascular_retained_fraction"]:.3f}/'
        f'{epoch_means["occlusion_retained_fraction"]:.3f}/'
        f'{epoch_means["clinical_retained_fraction"]:.3f}; '
        'mean |gate| context/q/e/b: '
        f'{epoch_means["context_gate_absolute_mean"]:.5f}/'
        f'{epoch_means["quarter_gate_absolute_mean"]:.5f}/'
        f'{epoch_means["eighth_gate_absolute_mean"]:.5f}/'
        f'{epoch_means["bottleneck_gate_absolute_mean"]:.5f}'
    )
    printable_bins = [
        f'{name}={value:.4f}'
        for name, value in validation_result['lesion_bin_mean_dice'].items()
        if math.isfinite(value)
    ]
    print('  Mean Dice by lesion bin: ' + ', '.join(printable_bins))

    atomic_write_json({
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'status': 'running',
        'last_completed_epoch': epoch,
        'best_positive_mean_dice': best_mean_dice,
        'best_epoch': best_epoch,
        'best_threshold': best_threshold,
        'validations_without_improvement': validations_without_improvement,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
    }, PILOT_STATE_PATH)
    if (
        epoch >= EARLY_STOPPING_MINIMUM_EPOCH
        and validations_without_improvement >= PATIENCE_VALIDATIONS
    ):
        stop_reason = 'early_stopping'
        print(
            f'Early stopping after {validations_without_improvement} '
            'held-out validations without sufficient improvement.'
        )
        break

completed_epoch = int(history_rows[-1]['epoch']) if history_rows else start_epoch - 1
pilot_complete = bool(
    completed_epoch >= MAX_EPOCHS or stop_reason in {
        'early_stopping', 'already_complete_on_resume'
    }
)
result_gate = {
    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    'pilot_version': PILOT_VERSION,
    'code_revision': PILOT_CODE_REVISION,
    'pilot_complete': pilot_complete,
    'stop_reason': stop_reason,
    'last_completed_epoch': completed_epoch,
    'best_positive_mean_dice': best_mean_dice,
    'best_epoch': best_epoch,
    'best_threshold': best_threshold,
    'fit_subjects': len(fold0_fit_df),
    'heldout_subjects': len(fold0_heldout_df),
    'isolated_construction_output_root': str(OUTPUT_ROOT),
    'isolated_pilot_output_root': str(PILOT_ROOT),
    'parameters': EXPECTED_V2_PARAMETERS,
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'outcome_workbooks_read': False,
    'pilot_review_required': True,
    'full_fivefold_training_authorized': False,
    'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
    'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
    'history_path': str(PILOT_HISTORY_PATH),
}
atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
atomic_write_json({
    **result_gate,
    'status': 'complete' if pilot_complete else 'interrupted',
}, PILOT_STATE_PATH)

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2.1 FOLD-0 PILOT COMPLETE')
print('=' * 80)
print(f'Stop reason:                        {stop_reason}')
print(f'Last completed epoch:               {completed_epoch}')
print(f'Best EMA positive mean Dice:        {best_mean_dice:.4f}')
print(f'Best epoch:                         {best_epoch}')
print(f'Calibrated threshold:               {best_threshold:.2f}')
print('Validation cadence:                 every epoch')
print(f'Development targets read:           {len(pilot_target_df)} (train split only)')
print('Locked validation targets read:     False')
print('Test targets read:                  False')
print(f'Architecture contract SHA-256:      {architecture_contract_sha256}')
print(f'Isolated V2.1 output root:           {PILOT_ROOT}')
print('Full five-fold training authorized: False (pilot review required)')
print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
print(f'Last checkpoint:                    {PILOT_LAST_CHECKPOINT_PATH}')
print(f'History:                            {PILOT_HISTORY_PATH}')
print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


In [ ]:
import sys
import os
import gc
import copy
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')
scipy = ensure_package('scipy')
matplotlib = ensure_package('matplotlib')

import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

plt.switch_backend('Agg')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AVAILABILITY_VERSION = 'v1_feature_availability_and_test_time_contract'
PREFLIGHT_VERSION = 'v2_2_diagnostic_model_reconstruction'
CODE_REVISION = 'r4_v2_2_read_only_failure_diagnostic'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_BASELINE_PARAMETERS = 15_052_628

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

AVAILABILITY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / AVAILABILITY_VERSION
)
AVAILABILITY_GATE_PATH = AVAILABILITY_ROOT / 'feature_availability_gate.json'
INPUT_CONTRACT_PATH = AVAILABILITY_ROOT / 'architecture_input_contract.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = OUTPUT_ROOT / 'architecture_config.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_model_construction_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_model_construction_audit.csv'

SEED = 20260927
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_428_468
PARAMETER_BUDGET_MINIMUM = 15_300_000
PARAMETER_BUDGET_MAXIMUM = 15_650_000

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return to_builtin(value.tolist())
    if isinstance(value, np.generic):
        return to_builtin(value.item())
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    AVAILABILITY_GATE_PATH,
    INPUT_CONTRACT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
availability_gate = json.loads(
    AVAILABILITY_GATE_PATH.read_text(encoding='utf-8')
)
input_contract = json.loads(INPUT_CONTRACT_PATH.read_text(encoding='utf-8'))
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')

if availability_gate.get('feature_availability_audit_passed') is not True:
    raise RuntimeError('Vascular-fusion feature-availability gate is not passed.')
if availability_gate.get('architecture_implementation_authorized') is not True:
    raise RuntimeError('Feature audit does not authorize architecture construction.')
if availability_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Feature audit must leave model training locked.')
if availability_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Feature audit did not remain target-free.')
if availability_gate.get('outcome_workbooks_read') is not False:
    raise RuntimeError('Feature audit read outcome workbooks.')
if availability_gate.get('manifest_sha256') != EXPECTED_TARGET_POLICY_MANIFEST_SHA256:
    raise RuntimeError('Feature audit target-policy manifest hash mismatch.')
if availability_gate.get('clinical_token_input_dimension') != CLINICAL_INPUT_DIM:
    raise RuntimeError('Clinical-token input dimension changed.')
if input_contract.get('architecture_version') != MODEL_VERSION:
    raise RuntimeError('Architecture input-contract version mismatch.')
if input_contract.get('model_training_authorized') is not False:
    raise RuntimeError('Input contract must leave model training locked.')
if input_contract.get('anti_leakage', {}).get('target_arrays_read') is not False:
    raise RuntimeError('Input contract did not remain target-free.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print(f'Trial8 v2 model preflight revision: {CODE_REVISION}')
print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Vascular-fusion availability gate: passed')
print('Target arrays read: False')
print('Outcome workbooks read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

atomic_write_npz(
    SCALER_NPZ_PATH,
    territory_feature_names=np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    territory_mean=territory_mean,
    territory_std=territory_std,
    territory_count=territory_count,
    vessel_feature_names=np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    vessel_source_indices=np.asarray(vessel_source_indices, dtype=np.int16),
    vessel_mean=vessel_mean,
    vessel_std=vessel_std,
    vessel_count=vessel_count,
    case_feature_names=np.asarray(MODEL_CASE_FEATURE_NAMES),
    case_source_indices=np.asarray(case_source_indices, dtype=np.int16),
    case_scale_mask=case_scale_mask.astype(np.uint8),
    case_mean=case_mean,
    case_std=case_std,
    case_count=case_count,
    clip_value=np.asarray(SCALER_CLIP, dtype=np.float32),
    fit_split=np.asarray('train'),
    fit_subject_count=np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    fit_subject_hash=np.asarray(scaler_fit_subject_hash),
    target_arrays_read=np.asarray(False, dtype=np.uint8),
)
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(scaler_record, SCALER_JSON_PATH)

print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    

    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(
                math.atanh(float(fraction_of_bound))
            )


class AsymmetricVascularStream3D(nn.Module):
    

    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        pooled = v5.mean(dim=(2, 3, 4))
        return v3, v4, v5, pooled


class SpatialConfidenceFusion3D(nn.Module):
    

    def __init__(
        self,
        vascular_channels,
        image_channels,
        confidence_dim=4,
        bound=0.1,
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, kernel_size=1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound
        )

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature,
                size=image_feature.shape[2:],
                mode='trilinear',
                align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.value_dim = int(value_dim)
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        safe_values = values * valid
        return self.encoder(torch.cat([safe_values, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    

    def __init__(
        self,
        vascular_dim,
        occlusion_dim,
        clinical_dim,
        output_dim=64,
        confidence_dim=4,
        bound=0.1,
        dropout=0.10,
    ):
        super().__init__()
        source_dim = vascular_dim + occlusion_dim + clinical_dim
        self.source = nn.Sequential(
            nn.Linear(source_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self,
        graph_context,
        vascular_context,
        occlusion_context,
        clinical_context,
        confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context,
            occlusion_context,
            clinical_context,
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
        vascular_channels=(8, 16, 32, 48),
        clinical_input_dim=44,
        clinical_embed_dim=32,
        occlusion_value_dim=7,
        occlusion_embed_dim=32,
        confidence_dim=4,
        fusion_bound=0.1,
        vascular_dropout=0.15,
        clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)

        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        vc1, vc2, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vascular_dim=vc4,
            occlusion_dim=occlusion_embed_dim,
            clinical_dim=clinical_embed_dim,
            output_dim=graph_hidden,
            confidence_dim=confidence_dim,
            bound=fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        modules = [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]
        for module in modules:
            module.activate_for_gradient_test(0.5)

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        vascular_image,
        confidence_features,
        occlusion_values,
        occlusion_valid,
        clinical_input,
        vascular_available,
        occlusion_available,
        clinical_available,
        force_atlas_dropout=None,
        force_vascular_dropout=None,
        force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available,
            self.vascular_dropout,
            force_vascular_dropout,
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available,
            self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available,
            self.clinical_dropout,
            force_clinical_dropout,
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        vascular_gate = effective_vascular.to(image.dtype)
        safe_vascular_image = (
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        v3, v4, v5, vascular_context = self.vascular_stream(
            safe_vascular_image
        )
        vascular_context = (
            vascular_context * vascular_gate[:, None]
        )
        safe_occlusion_valid = (
            occlusion_valid
            * effective_occlusion[:, None].to(occlusion_valid.dtype)
        )
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input)
        clinical_context = (
            clinical_context * effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context,
            vascular_context,
            occlusion_context,
            clinical_context,
            effective_confidence,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')


preflight_row = None
for _, candidate_row in preflight_candidates.iterrows():
    candidate_npz_path = Path(str(candidate_row['vascufate_v2_npz_path']))
    with np.load(candidate_npz_path, allow_pickle=False) as candidate_data:
        candidate_lvo_present = (
            'lvo_present' in candidate_data.files
            and bool(np.asarray(candidate_data['lvo_present']).reshape(-1)[0])
        )
    if candidate_lvo_present:
        preflight_row = candidate_row
        break
if preflight_row is None:
    raise RuntimeError(
        'No atlas-enabled train-only case has a valid LVO source for the '
        'complete v2 branch smoke test.'
    )
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    required_npz_inputs = {
        'image',
        'subject_id',
        'input_channels',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
    }
    missing_npz_inputs = sorted(required_npz_inputs - set(data.files))
    if missing_npz_inputs:
        raise RuntimeError(
            'Preflight source NPZ lacks required input arrays: '
            + ', '.join(missing_npz_inputs)
        )
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
    input_channels = decode_strings(data['input_channels'])
    cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
    lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
    lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')
if input_channels != EXPECTED_INPUT_CHANNELS:
    raise RuntimeError('Preflight input-channel schema changed.')
if cow_soft_union.shape != image.shape[1:]:
    raise RuntimeError('CoW prior shape differs from the image grid.')
if lvo_proximity.shape != image.shape[1:]:
    raise RuntimeError('LVO proximity shape differs from the image grid.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

cow_prior_valid = bool(
    case_features_source[
        SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')
    ] > 0.5
)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))
cow_patch = center_crop_or_pad(
    cow_soft_union, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
lvo_patch = center_crop_or_pad(
    lvo_proximity, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
vascular_patch = np.stack([
    image_patch[input_channels.index('cta')],
    image_patch[input_channels.index('tmax')],
    image_patch[input_channels.index('perfusion_valid')],
    cow_patch,
    lvo_patch,
], axis=0).astype(np.float32)




clinical_input = np.zeros(CLINICAL_INPUT_DIM, dtype=np.float32)
clinical_input[:22] = np.linspace(-0.5, 0.5, 22, dtype=np.float32)
clinical_input[22:] = (np.arange(22) % 3 == 0).astype(np.float32)

occlusion_feature_names = [
    'cta_mean',
    'cta_p90',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
occlusion_values = [float(lvo_present), float(lvo_patch.max())]
occlusion_valid = [1.0, float(lvo_present)]
for feature_name in occlusion_feature_names:
    feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
    valid_nodes = selected_vessel_valid[:, feature_index]
    if bool(valid_nodes.any()):
        occlusion_values.append(float(scaled_vessel[valid_nodes, feature_index].mean()))
        occlusion_valid.append(1.0)
    else:
        occlusion_values.append(0.0)
        occlusion_valid.append(0.0)
occlusion_values = np.asarray(occlusion_values, dtype=np.float32)
occlusion_valid = np.asarray(occlusion_valid, dtype=np.float32)
if occlusion_values.shape != (OCCLUSION_VALUE_DIM,):
    raise RuntimeError('Occlusion-source vector dimension changed.')

confidence_features = np.asarray([
    float(vascular_patch[2].mean()),
    float(cow_prior_valid),
    float(np.clip(selected_case[1], 0.0, 1.0)),
    float(lvo_present),
], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
    'vascular_image': torch.from_numpy(vascular_patch[None]).to(device),
    'confidence_features': torch.from_numpy(
        confidence_features[None]
    ).to(device),
    'occlusion_values': torch.from_numpy(occlusion_values[None]).to(device),
    'occlusion_valid': torch.from_numpy(occlusion_valid[None]).to(device),
    'clinical_input': torch.from_numpy(clinical_input[None]).to(device),
    'vascular_available': torch.ones(1, dtype=torch.bool, device=device),
    'occlusion_available': torch.as_tensor(
        [lvo_present], dtype=torch.bool, device=device
    ),
    'clinical_available': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
vascular_stream_parameters = parameter_count(model.vascular_stream)
spatial_fusion_parameters = parameter_count(
    model.vascular_fusion_quarter
) + parameter_count(model.vascular_fusion_eighth) + parameter_count(
    model.vascular_fusion_bottleneck
)
clinical_parameters = parameter_count(model.clinical_encoder)
occlusion_parameters = parameter_count(model.occlusion_encoder)
context_fusion_parameters = parameter_count(model.context_fusion)
new_fusion_parameters = (
    vascular_stream_parameters
    + spatial_fusion_parameters
    + clinical_parameters
    + occlusion_parameters
    + context_fusion_parameters
)
backbone_parameters = total_parameters - (
    graph_parameters
    + film_parameters
    + head_parameters
    + new_fusion_parameters
)
preserved_v1_parameters = (
    backbone_parameters + graph_parameters + film_parameters + head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'asymmetric vascular stream', 'parameters': vascular_stream_parameters},
    {'component': 'confidence-gated spatial fusion', 'parameters': spatial_fusion_parameters},
    {'component': 'clinical token encoder', 'parameters': clinical_parameters},
    {'component': 'occlusion source encoder', 'parameters': occlusion_parameters},
    {'component': 'gated context fusion', 'parameters': context_fusion_parameters},
    {'component': 'preserved v1 path', 'parameters': preserved_v1_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad():
    normal_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    atlas_dropped_output = model(
        **batch,
        force_atlas_dropout=True,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    new_sources_dropped_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=True,
        force_clinical_dropout=True,
    )

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
    'fused_global_context': (1, GRAPH_HIDDEN),
    'new_source_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(new_sources_dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        atlas_dropped_output['atlas_dropped'].item()
        and not atlas_dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'],
        atlas_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'initial_segmentation_is_new_sources_neutral': torch.allclose(
        normal_output['logits'],
        new_sources_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
    'zero_init_new_fusion_gates_are_zero': all(
        torch.count_nonzero(module.network[-1].weight).item() == 0
        and torch.count_nonzero(module.network[-1].bias).item() == 0
        for module in [
            model.vascular_fusion_quarter.gate,
            model.vascular_fusion_eighth.gate,
            model.vascular_fusion_bottleneck.gate,
            model.context_fusion.gate,
        ]
    ),
    'normal_vascular_source_enabled': bool(
        normal_output['effective_vascular_enabled'].item()
    ),
    'normal_clinical_source_enabled': bool(
        normal_output['effective_clinical_enabled'].item()
    ),
    'forced_new_source_dropout_applied': bool(
        new_sources_dropped_output['vascular_dropped'].item()
        and new_sources_dropped_output['occlusion_dropped'].item()
        and new_sources_dropped_output['clinical_dropped'].item()
    ),
})




model.activate_new_fusion_gates_for_gradient_test()
with torch.no_grad():
    nn.init.normal_(model.global_film.project[-1].weight, mean=0.0, std=1e-3)
    nn.init.zeros_(model.global_film.project[-1].bias)

model.eval()
with torch.no_grad():
    activated_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    vascular_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    occlusion_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=True,
        force_clinical_dropout=False,
    )
    clinical_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=True,
    )

ablation_differences = {
    'vascular_max_abs_logit_change': float(
        (activated_output['logits'] - vascular_ablation_output['logits'])
        .abs().max().item()
    ),
    'occlusion_max_abs_logit_change': float(
        (activated_output['logits'] - occlusion_ablation_output['logits'])
        .abs().max().item()
    ),
    'clinical_max_abs_logit_change': float(
        (activated_output['logits'] - clinical_ablation_output['logits'])
        .abs().max().item()
    ),
}
ablation_checks = {
    'activated_outputs_finite': all(
        torch.isfinite(activated_output[name]).all().item()
        for name in expected_shapes
    ),
    'vascular_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['vascular_max_abs_logit_change'] > 1e-8
    ),
    'occlusion_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['occlusion_max_abs_logit_change'] > 1e-8
    ),
    'clinical_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['clinical_max_abs_logit_change'] > 1e-8
    ),
    'all_activated_gates_remain_bounded': all(
        float(activated_output[name].abs().max().item()) <= FUSION_BOUND + 1e-6
        for name in [
            'context_fusion_gate',
            'vascular_quarter_gate',
            'vascular_eighth_gate',
            'vascular_bottleneck_gate',
        ]
    ),
}

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=False,
):
    gradient_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('vascular_stream', model.vascular_stream),
    ('spatial_fusion', nn.ModuleList([
        model.vascular_fusion_quarter,
        model.vascular_fusion_eighth,
        model.vascular_fusion_bottleneck,
    ])),
    ('clinical_encoder', model.clinical_encoder),
    ('occlusion_encoder', model.occlusion_encoder),
    ('context_fusion', model.context_fusion),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'vascular_stream_has_gradients_after_gate_activation': (
        gradient_norms['vascular_stream'] > 0
    ),
    'spatial_fusion_has_gradients_after_gate_activation': (
        gradient_norms['spatial_fusion'] > 0
    ),
    'clinical_encoder_has_gradients_after_gate_activation': (
        gradient_norms['clinical_encoder'] > 0
    ),
    'occlusion_encoder_has_gradients_after_gate_activation': (
        gradient_norms['occlusion_encoder'] > 0
    ),
    'context_fusion_has_gradients_after_gate_activation': (
        gradient_norms['context_fusion'] > 0
    ),
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'image_channels': int(image.shape[0]),
    'patch_size': PATCH_SIZE,
    'backbone_channels': [32, 64, 128, 224, 320],
    'baseline_architecture_preserved': True,
    'baseline_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    'graph_variant': 'two_layer_relation_aware_gated_GATv2',
    'graph_hidden': GRAPH_HIDDEN,
    'graph_heads': GRAPH_HEADS,
    'graph_layers': GRAPH_LAYERS,
    'relation_bases': RELATION_BASES,
    'graph_dropout': GRAPH_DROPOUT,
    'edge_weight_use': 'log_attention_prior',
    'territory_nodes': NUM_TERRITORY_NODES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'conditioning': (
        'preserved_bounded_zero_initialized_multiscale_FiLM_plus_'
        'confidence_gated_zero_initialized_vascular_context_fusion'
    ),
    'film_bound': FILM_BOUND,
    'fusion_bound': FUSION_BOUND,
    'asymmetric_vascular_stream': {
        'input_channels': [
            'cta', 'tmax', 'perfusion_valid',
            'cow_soft_union', 'lvo_proximity',
        ],
        'channels': VASCULAR_CHANNELS,
        'fusion_scales': ['quarter', 'eighth', 'sixteenth'],
        'initial_gate': 'exact_zero',
    },
    'clinical_token': {
        'input_dimension': CLINICAL_INPUT_DIM,
        'embedding_dimension': CLINICAL_EMBED_DIM,
        'values': 22,
        'paired_missingness_indicators': 22,
        'fold_specific_scaler_required_before_training': True,
    },
    'occlusion_source': {
        'value_dimension': OCCLUSION_VALUE_DIM,
        'embedding_dimension': OCCLUSION_EMBED_DIM,
        'validity_masked': True,
        'uses_tmax_not_ttp': True,
    },
    'confidence_features': [
        'perfusion_valid_fraction',
        'cow_prior_valid',
        'registration_confidence',
        'lvo_present',
    ],
    'staged_neutrality': {
        'new_fusion_gates_exactly_zero_at_initialization': True,
        'controlled_gate_activation_used_only_for_preflight': True,
        'saved_or_trained_weights_created': False,
    },
    'graph_to_logit_residual': False,
    'expert_router': False,
    'deep_supervision': True,
    'territory_auxiliary_head': True,
    'scaler_policy': 'fit_on_train_only_then_freeze',
    'expected_total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
        'asymmetric_vascular_stream': vascular_stream_parameters,
        'spatial_fusion': spatial_fusion_parameters,
        'clinical_encoder': clinical_parameters,
        'occlusion_encoder': occlusion_parameters,
        'context_fusion': context_fusion_parameters,
        'new_fusion_total': new_fusion_parameters,
        'preserved_v1_total': preserved_v1_parameters,
    },
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_sha256': scaler_sha256,
    'feature_availability_gate_path': str(AVAILABILITY_GATE_PATH),
    'architecture_input_contract_path': str(INPUT_CONTRACT_PATH),
    'ablation_differences': ablation_differences,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'model_training_authorized': False,
}
atomic_write_json(architecture_config, ARCHITECTURE_JSON_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **forward_checks,
    **ablation_checks,
    **gradient_checks,
    'feature_availability_audit_passed': (
        availability_gate.get('feature_availability_audit_passed') is True
    ),
    'input_contract_authorizes_architecture_implementation': (
        input_contract.get('architecture_implementation_authorized') is True
    ),
    'preserved_v1_parameter_count_matches': (
        preserved_v1_parameters == EXPECTED_BASELINE_PARAMETERS
    ),
    'parameter_count_in_frozen_efficiency_budget': (
        PARAMETER_BUDGET_MINIMUM
        <= total_parameters
        <= PARAMETER_BUDGET_MAXIMUM
    ),
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'new_fusion_parameter_count_is_bounded': (
        0 < new_fusion_parameters < 600_000
    ),
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'direct_graph_to_logit_residual_disabled': True,
    'expert_routing_disabled': True,
    'training_remained_locked': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'ablation', 'check': name, 'passed': passed}
    for name, passed in ablation_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
 ] + [
    {'category': 'final_gate', 'check': name, 'passed': passed}
    for name, passed in preflight_checks.items()
    if name not in {
        *manifest_checks.keys(),
        *scaler_checks.keys(),
        *forward_checks.keys(),
        *ablation_checks.keys(),
        *gradient_checks.keys(),
    }
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'ablation_differences': ablation_differences,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'controlled_gate_activation_was_diagnostic_only': True,
    'saved_or_trained_weights_created': False,
    'v2_loader_and_clinical_scaler_implementation_authorized': preflight_passed,
    'model_training_authorized': False,
}
atomic_write_json(gate_record, PREFLIGHT_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 MODEL CONSTRUCTION PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'New fusion parameters:              {new_fusion_parameters:,}')
print(f'  Asymmetric vascular stream:       {vascular_stream_parameters:,}')
print(f'  Spatial confidence fusion:        {spatial_fusion_parameters:,}')
print(f'  Clinical encoder:                 {clinical_parameters:,}')
print(f'  Occlusion encoder:                {occlusion_parameters:,}')
print(f'  Context fusion:                   {context_fusion_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('New fusion initially neutral:       True')
print('Controlled ablations responsive:    True')
print('Graph scaler fitted on:             train only (102 subjects)')
print('Clinical scaler fitted:             False (next preflight)')
print('Target arrays read:                 False')
print('Outcome workbooks read:              False')
print('Model training authorized:          False')
print(f'V2 loader implementation allowed:   {preflight_passed}')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print(
    '\nThe standalone block will now verify the completed multimodal-loader '
    'gate and continue into the isolated fold-0 pilot.'
)






PILOT_CODE_REVISION = 'r1_branch_component_threshold_overfit_audit'
PILOT_VERSION = 'v2_2_fold0_failure_diagnostic_r1'
LOADER_PREFLIGHT_VERSION = 'v1_fold_clinical_scalers_multimodal_loader'
FIVEFOLD_PREFLIGHT_VERSION = 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_V2_PARAMETERS = 15_428_468
HELDOUT_FOLD = 0
PILOT_SEED = 20260928
PILOT_PATCH_SIZE = (128, 128, 64)
NUMBER_OF_FOLDS = 5
MINIMUM_POSITIVE_PATCHES_PER_BIN = 20
MAXIMUM_NEGATIVE_CENTER_ATTEMPTS = 512
MAXIMUM_CENTER_RETRIES = 16
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
PILOT_CLINICAL_FEATURE_NAMES = [
    'sex_female',
    'age_years',
    'atrial_fibrillation',
    'hypertension',
    'diabetes',
    'hyperlipidemia',
    'anticoagulation',
    'lipid_lowering_drugs',
    'platelet_aggregation_inhibitors',
    'glucose_mg_dl',
    'leucocytes_g_l',
    'crp_mg_dl',
    'inr',
    'wake_up_stroke',
    'in_house_stroke',
    'external_referral',
    'onset_to_door_hours',
    'alert_to_door_hours',
    'nihss_at_admission',
    'mrs_at_admission',
    'mrs_premorbid',
    'door_to_imaging_hours',
]

MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
LOADER_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / LOADER_PREFLIGHT_VERSION
)
LOADER_GATE_PATH = LOADER_PREFLIGHT_ROOT / 'v2_multimodal_loader_gate.json'
FIVEFOLD_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / FIVEFOLD_PREFLIGHT_VERSION
)
FIVEFOLD_GATE_PATH = (
    FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)
FIVEFOLD_POLICY_PATH = FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_training_policy.json'

PILOT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / MODEL_VERSION
    / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_START_GATE_PATH = PILOT_ROOT / 'fold0_pilot_start_gate.json'
PILOT_CONFIG_PATH = PILOT_ROOT / 'fold0_pilot_config.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'fold0_training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'fold0_pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'fold0_pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'fold0_epoch1_schedule_audit.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_json_contract_hash(record, excluded_top_level_keys=()):
    excluded = set(excluded_top_level_keys)
    stable_record = {
        key: value for key, value in record.items() if key not in excluded
    }
    payload = json.dumps(
        to_builtin(stable_record),
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


pilot_required_paths = [
    MULTIMODAL_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    LOADER_GATE_PATH,
    FIVEFOLD_GATE_PATH,
    FIVEFOLD_POLICY_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    ARCHITECTURE_JSON_PATH,
    PREFLIGHT_GATE_PATH,
]
pilot_missing_paths = [str(path) for path in pilot_required_paths if not path.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Fold-0 pilot prerequisites are missing:\n' + '\n'.join(pilot_missing_paths)
    )

multimodal_manifest_sha256 = file_sha256(MULTIMODAL_MANIFEST_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)
loader_gate_sha256 = file_sha256(LOADER_GATE_PATH)
fivefold_gate_sha256 = file_sha256(FIVEFOLD_GATE_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)
v2_model_gate_sha256 = file_sha256(PREFLIGHT_GATE_PATH)
architecture_contract_sha256 = pilot_json_contract_hash(
    architecture_config,
    excluded_top_level_keys={'created_at_utc', 'ablation_differences'},
)
if multimodal_manifest_sha256 != EXPECTED_MULTIMODAL_MANIFEST_SHA256:
    raise RuntimeError('Multimodal candidate manifest hash changed.')
if fold_manifest_sha256 != EXPECTED_FOLD_MANIFEST_SHA256:
    raise RuntimeError('Five-fold manifest hash changed.')

loader_gate = json.loads(LOADER_GATE_PATH.read_text(encoding='utf-8'))
fivefold_gate = json.loads(FIVEFOLD_GATE_PATH.read_text(encoding='utf-8'))
fivefold_policy = json.loads(FIVEFOLD_POLICY_PATH.read_text(encoding='utf-8'))
v2_model_gate = json.loads(PREFLIGHT_GATE_PATH.read_text(encoding='utf-8'))

pilot_upstream_checks = {
    'v2_model_preflight_passed': v2_model_gate.get('preflight_passed') is True,
    'v2_model_parameter_count_matches': int(
        v2_model_gate.get('total_parameters', -1)
    ) == EXPECTED_V2_PARAMETERS,
    'v2_model_architecture_hash_matches': (
        v2_model_gate.get('architecture_config_sha256') == architecture_sha256
    ),
    'multimodal_loader_preflight_passed': (
        loader_gate.get('preflight_passed') is True
    ),
    'multimodal_manifest_hash_matches_loader_gate': (
        loader_gate.get('candidate_manifest_sha256')
        == multimodal_manifest_sha256
    ),
    'fold_manifest_hash_matches_loader_gate': (
        loader_gate.get('fold_manifest_sha256') == fold_manifest_sha256
    ),
    'graph_schema_hash_matches_loader_gate': (
        loader_gate.get('graph_schema_sha256') == safe_schema_sha256
    ),
    'pilot_implementation_authorized': (
        loader_gate.get('pilot_training_implementation_authorized') is True
    ),
    'upstream_training_remained_locked': (
        loader_gate.get('fold0_pilot_training_authorized') is False
        and loader_gate.get('full_fivefold_training_authorized') is False
    ),
    'loader_target_free': loader_gate.get('target_arrays_read') is False,
    'loader_outcome_free': loader_gate.get('outcome_workbooks_read') is False,
    'fivefold_loader_loss_sampler_passed': (
        fivefold_gate.get('preflight_passed') is True
        and fivefold_gate.get(
            'baseline_loader_loss_sampler_contract_verified'
        ) is True
    ),
    'fivefold_policy_training_locked': (
        fivefold_gate.get('fold0_pilot_training_authorized') is False
        and fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed_upstream': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed_upstream': fivefold_gate.get('test_targets_read') is False,
    'architecture_still_has_no_expert_router': (
        architecture_config.get('expert_router') is False
    ),
    'architecture_still_has_no_graph_logit_residual': (
        architecture_config.get('graph_to_logit_residual') is False
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [name for name, passed in pilot_upstream_checks.items() if not passed]
    raise RuntimeError('Fold-0 pilot upstream checks failed: ' + ', '.join(failed))

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].astype(str).tolist() != pilot_fold_df[
    'subject_id'
].astype(str).tolist():
    raise RuntimeError('Multimodal and fold manifests have different subjects.')
if 'development_fold' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_fold'] = pilot_fold_df[
        'development_fold'
    ].astype(int)
if 'development_role' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_role'] = pilot_fold_df[
        'development_role'
    ].astype(str)

pilot_required_columns = {
    'subject_id', 'dev_split', 'development_fold', 'vascufate_v2_npz_path',
    'trial8_npz_sha256', 'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
    'trial8_clinical_sidecar_sha256',
    'trial8_v2_multimodal_loader_preflight_passed',
    'trial8_model_training_authorized', 'trial8_supervision_mode',
    'trial8_effective_target_volume_ml',
    'trial8_effective_lesion_volume_bin', 'trial8_sampling_positive',
    'trial8_hard_metric_eligible',
}
pilot_missing_columns = sorted(
    pilot_required_columns - set(pilot_manifest_df.columns)
)
if pilot_missing_columns:
    raise RuntimeError(
        'Multimodal manifest is missing columns: ' + ', '.join(pilot_missing_columns)
    )

development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'].astype(int) != HELDOUT_FOLD
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'].astype(int) == HELDOUT_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()

pilot_manifest_checks = {
    'manifest_has_148_subjects': len(pilot_manifest_df) == 148,
    'manifest_subjects_unique': pilot_manifest_df['subject_id'].nunique() == 148,
    'split_counts_match': pilot_manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'loader_preflight_passed_for_all': bool(
        pilot_manifest_df[
            'trial8_v2_multimodal_loader_preflight_passed'
        ].map(as_bool).all()
    ),
    'manifest_training_remained_locked': not bool(
        pilot_manifest_df['trial8_model_training_authorized'].map(as_bool).any()
    ),
    'fold0_fit_has_82_subjects': len(fold0_fit_df) == 82,
    'fold0_heldout_has_20_subjects': len(fold0_heldout_df) == 20,
    'locked_validation_has_23_subjects': len(locked_validation_df) == 23,
    'sealed_test_has_23_subjects': len(sealed_test_df) == 23,
    'fold0_fit_and_heldout_are_disjoint': not bool(
        set(fold0_fit_df['subject_id']) & set(fold0_heldout_df['subject_id'])
    ),
}
if not all(pilot_manifest_checks.values()):
    failed = [name for name, passed in pilot_manifest_checks.items() if not passed]
    raise RuntimeError('Fold-0 manifest checks failed: ' + ', '.join(failed))

fold0_graph_scaler_path = Path(loader_gate['graph_scaler_paths'][str(HELDOUT_FOLD)])
fold0_graph_scaler_sha256 = loader_gate['graph_scaler_sha256'][
    str(HELDOUT_FOLD)
]
fold0_clinical_scaler_path = Path(
    loader_gate['clinical_scaler_paths'][str(HELDOUT_FOLD)]
)
fold0_clinical_scaler_sha256 = loader_gate['clinical_scaler_sha256'][
    str(HELDOUT_FOLD)
]
for scaler_path, expected_hash, label in [
    (fold0_graph_scaler_path, fold0_graph_scaler_sha256, 'graph'),
    (fold0_clinical_scaler_path, fold0_clinical_scaler_sha256, 'clinical'),
]:
    if not scaler_path.exists():
        raise FileNotFoundError(f'Fold-0 {label} scaler is missing: {scaler_path}')
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-0 {label} scaler hash changed.')

with np.load(fold0_graph_scaler_path, allow_pickle=False) as data:
    fold0_graph_scaler = {name: np.asarray(data[name]) for name in data.files}
with np.load(fold0_clinical_scaler_path, allow_pickle=False) as data:
    fold0_clinical_scaler = {name: np.asarray(data[name]) for name in data.files}

if decode_strings(fold0_graph_scaler['territory_feature_names']) != (
    EXPECTED_NODE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 territory feature order changed.')
if decode_strings(fold0_graph_scaler['vessel_feature_names']) != (
    MODEL_VESSEL_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 vessel feature order changed.')
if decode_strings(fold0_graph_scaler['case_feature_names']) != (
    MODEL_CASE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 case feature order changed.')
if decode_strings(
    fold0_clinical_scaler['feature_names']
) != PILOT_CLINICAL_FEATURE_NAMES:
    raise RuntimeError('Fold-0 clinical feature order changed.')

fold0_fit_subject_hash = pilot_subject_hash(fold0_fit_df['subject_id'])
fold0_heldout_subject_hash = pilot_subject_hash(fold0_heldout_df['subject_id'])
pilot_scaler_contract_checks = {
    'graph_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_graph_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'graph_scaler_fit_count_is_82': int(
        np.asarray(fold0_graph_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'graph_scaler_fit_subject_hash_matches': scalar_string(
        fold0_graph_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'graph_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_graph_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'clinical_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_clinical_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'clinical_scaler_fit_count_is_82': int(
        np.asarray(fold0_clinical_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'clinical_scaler_heldout_count_is_20': int(
        np.asarray(fold0_clinical_scaler['heldout_subject_count']).reshape(-1)[0]
    ) == 20,
    'clinical_scaler_fit_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'clinical_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'graph_scaler_is_target_free': not bool(
        np.asarray(fold0_graph_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_target_free': not bool(
        np.asarray(fold0_clinical_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_outcome_free': not bool(
        np.asarray(
            fold0_clinical_scaler['outcome_workbooks_read']
        ).reshape(-1)[0]
    ),
}
if not all(pilot_scaler_contract_checks.values()):
    failed = [
        name for name, passed in pilot_scaler_contract_checks.items()
        if not passed
    ]
    raise RuntimeError('Fold-0 scaler contract failed: ' + ', '.join(failed))

print('\n' + '=' * 80)
print('TRIAL8 V2 FOLD-0 PILOT CONTRACT PASSED')
print('=' * 80)
print(f'Pilot code revision:                 {PILOT_CODE_REVISION}')
print(f'Fold-0 fit / held-out:               {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
print(f'Locked validation / sealed test:     {len(locked_validation_df)} / {len(sealed_test_df)}')
print('Target access authorized:            development train split only')
print('Locked validation targets read:      False')
print('Test targets read:                   False')


for disposable_name in [
    'model', 'batch', 'normal_output', 'forced_atlas_output',
    'forced_vascular_output', 'forced_occlusion_output',
    'forced_clinical_output', 'gradient_output', 'smoke_loss',
]:
    globals().pop(disposable_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(value) for value in mask.shape),
        'packed': np.packbits(mask.reshape(-1)),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    return np.unpackbits(record['packed'], count=count).reshape(
        record['shape']
    ).astype(bool)


fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')
all_development_row_by_subject = {
    str(row['subject_id']): row for row in development_df.to_dict('records')
}
pilot_mask_records = {}
pilot_shape_by_subject = {}
pilot_target_rows = []
for row in development_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required_target_arrays = {
            'target_binary', 'target_soft', 'binary_loss_weight',
            'crop_mask', 'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required_target_arrays - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        hard_weight = pilot_scalar(data, 'binary_loss_weight')
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
        source_volume_ml = pilot_scalar(data, 'source_target_volume_ml')
    if target_binary.shape != target_soft.shape or target_binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if not np.isfinite(target_soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target arrays are non-finite.')
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        sampling_support = target_binary
        expected_hard_weight = 1.0
    elif supervision_mode == 'soft_only_subvoxel':
        sampling_support = target_soft > 0
        expected_hard_weight = 0.0
    elif supervision_mode == 'true_empty':
        sampling_support = np.zeros_like(target_binary)
        expected_hard_weight = 1.0
    else:
        raise RuntimeError(f'{subject_id}: invalid supervision mode.')
    exclusion_support = target_binary | (target_soft > 0)
    pilot_shape_by_subject[subject_id] = tuple(
        int(value) for value in target_binary.shape
    )
    if not math.isclose(hard_weight, expected_hard_weight):
        raise RuntimeError(f'{subject_id}: hard-loss policy changed.')
    if subject_id in fit_row_by_subject:
        pilot_mask_records[subject_id] = {
            'sampling_support': pilot_pack_mask(sampling_support),
            'exclusion_support': pilot_pack_mask(exclusion_support),
            'crop_mask': pilot_pack_mask(crop_mask),
        }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_fold': int(row['development_fold']),
        'pilot_role': 'fit' if subject_id in fit_row_by_subject else 'heldout',
        'supervision_mode': supervision_mode,
        'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
        'sampling_support_voxels': int(sampling_support.sum()),
        'binary_voxels': int(target_binary.sum()),
        'soft_nonzero_voxels': int((target_soft > 0).sum()),
        'source_target_volume_ml': source_volume_ml,
        'hard_weight': hard_weight,
        'voxel_volume_ml': float(np.prod(spacing) / 1000.0),
        'target_arrays_read_split': 'train',
    })
pilot_target_df = pd.DataFrame(pilot_target_rows)
atomic_write_csv(pilot_target_df, PILOT_TARGET_AUDIT_PATH)
pilot_target_by_subject = pilot_target_df.set_index('subject_id').to_dict('index')

pilot_target_checks = {
    'exactly_102_development_targets_read': len(pilot_target_df) == 102,
    'target_access_roles_are_fit_or_heldout': set(
        pilot_target_df['pilot_role']
    ) == {'fit', 'heldout'},
    'target_access_is_train_split_only': set(
        pilot_target_df['target_arrays_read_split']
    ) == {'train'},
    'fit_sampling_masks_cover_82_subjects': len(pilot_mask_records) == 82,
    'soft_only_hard_loss_disabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'] == 'soft_only_subvoxel',
            'hard_weight',
        ].eq(0.0).all()
    ),
    'hard_and_empty_hard_loss_enabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'].isin(
                ['hard_and_soft', 'true_empty']
            ),
            'hard_weight',
        ].eq(1.0).all()
    ),
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
}
if not all(pilot_target_checks.values()):
    failed = [name for name, passed in pilot_target_checks.items() if not passed]
    raise RuntimeError('Fold-0 target checks failed: ' + ', '.join(failed))

territory_edge_index_numpy = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type_numpy = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)
PILOT_RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def pilot_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, PILOT_PATCH_SIZE):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def pilot_extract_patch(array, slices, spatial_axes):
    source = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source[axis] = spatial_slice
    cropped = array[tuple(source)]
    return center_crop_or_pad(cropped, PILOT_PATCH_SIZE, spatial_axes)


region_cache = {}
region_cache_order = []
REGION_CACHE_LIMIT = 24


def pilot_load_region_map(subject_id):
    if subject_id in region_cache:
        return region_cache[subject_id]
    row = all_development_row_by_subject[subject_id]
    shape = pilot_shape_by_subject[subject_id]
    if not as_bool(row['trial8_atlas_enabled']):
        region = np.zeros(shape, dtype=np.uint8)
    else:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region = np.asanyarray(atlas_image.dataobj)
        if region.ndim == 4 and region.shape[-1] == 1:
            region = region[..., 0]
        region = np.rint(region).astype(np.uint8)
        if region.shape != shape:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region.min() < 0 or region.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    region_cache[subject_id] = region
    region_cache_order.append(subject_id)
    if len(region_cache_order) > REGION_CACHE_LIMIT:
        evicted = region_cache_order.pop(0)
        region_cache.pop(evicted, None)
    return region


def pilot_topology_candidates(support, region_map):
    lesion_labels = set(
        int(value) for value in np.unique(region_map[support])
        if 1 <= int(value) <= 30
    )
    relation_priority = {2: 0, 1: 1, 3: 2}
    relation_by_label = {}
    for edge_index in range(territory_edge_index_numpy.shape[1]):
        source = int(territory_edge_index_numpy[0, edge_index]) + 1
        destination = int(territory_edge_index_numpy[1, edge_index]) + 1
        relation = int(territory_edge_type_numpy[edge_index])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in relation_priority
        ):
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    labels = sorted(
        relation_by_label,
        key=lambda label: (relation_priority[relation_by_label[label]], label),
    )
    return labels, relation_by_label


def pilot_sample_coordinates(mask, rng):
    flat = np.flatnonzero(mask)
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > MAXIMUM_NEGATIVE_CENTER_ATTEMPTS:
        flat = rng.choice(
            flat, size=MAXIMUM_NEGATIVE_CENTER_ATTEMPTS, replace=False
        )
    else:
        flat = flat.copy()
        rng.shuffle(flat)
    return np.column_stack(np.unravel_index(flat, mask.shape)).astype(np.int64)


def pilot_zero_support_center(exclusion, candidate, rng):
    for center in pilot_sample_coordinates(candidate, rng):
        slices = pilot_patch_slices(exclusion.shape, center)
        if not exclusion[slices].any():
            return center, slices
    return None, None


def pilot_positive_center(subject_id, rng, epoch):
    support = pilot_unpack_mask(
        pilot_mask_records[subject_id]['sampling_support']
    )
    if not support.any():
        raise RuntimeError(f'{subject_id}: positive schedule has no support.')
    if pilot_target_by_subject[subject_id]['supervision_mode'] == (
        'soft_only_subvoxel'
    ):
        labels, component_count = ndimage.label(
            support, structure=np.ones((3, 3, 3), dtype=np.uint8)
        )
        selected_component = 1 + ((epoch - 1) % component_count)
        coordinates = np.argwhere(labels == selected_component)
    else:
        coordinates = np.argwhere(support)
    seed = coordinates[int(rng.integers(0, len(coordinates)))].astype(np.int64)
    for _ in range(MAXIMUM_CENTER_RETRIES):
        jitter = np.asarray([
            int(rng.integers(-12, 13)),
            int(rng.integers(-12, 13)),
            int(rng.integers(-4, 5)),
        ])
        center = np.clip(seed + jitter, 0, np.asarray(support.shape) - 1)
        slices = pilot_patch_slices(support.shape, center)
        if support[slices].any():
            return center, slices
    slices = pilot_patch_slices(support.shape, seed)
    if not support[slices].any():
        raise RuntimeError(f'{subject_id}: failed to retain positive support.')
    return seed, slices


def pilot_balanced_positive_schedule(rng):
    schedule = []
    for lesion_bin in POSITIVE_BINS:
        pool = fold0_fit_df.loc[
            (fold0_fit_df['trial8_effective_lesion_volume_bin'] == lesion_bin)
            & fold0_fit_df['trial8_sampling_positive'].map(as_bool),
            'subject_id',
        ].astype(str).tolist()
        if not pool:
            raise RuntimeError(f'Fold-0 positive bin {lesion_bin} is empty.')
        target_count = max(MINIMUM_POSITIVE_PATCHES_PER_BIN, len(pool))
        selected = pool.copy()
        hard_pool = [
            subject_id for subject_id in pool
            if pilot_target_by_subject[subject_id]['supervision_mode']
            == 'hard_and_soft'
        ]
        rng.shuffle(hard_pool)
        if not hard_pool and len(selected) < target_count:
            raise RuntimeError(f'No hard-target repeats for {lesion_bin}.')
        repeat_index = 0
        while len(selected) < target_count:
            selected.append(hard_pool[repeat_index % len(hard_pool)])
            repeat_index += 1
        rng.shuffle(selected)
        schedule.extend(selected)
    rng.shuffle(schedule)
    return schedule


def pilot_build_epoch_schedule(epoch):
    rng = np.random.default_rng(PILOT_SEED + epoch * 1_000_003)
    positive_subjects = pilot_balanced_positive_schedule(rng)
    positive_count = len(positive_subjects)
    topology_count = int(math.ceil(positive_count / 4.0))
    brain_count = int(math.floor(positive_count / 4.0))
    topology_pool = fold0_fit_df.loc[
        fold0_fit_df['trial8_atlas_enabled'].map(as_bool)
        & fold0_fit_df['trial8_sampling_positive'].map(as_bool)
        & fold0_fit_df['trial8_supervision_mode'].eq('hard_and_soft'),
        'subject_id',
    ].astype(str).tolist()
    brain_pool = fold0_fit_df.loc[
        ~fold0_fit_df['trial8_supervision_mode'].eq('soft_only_subvoxel'),
        'subject_id',
    ].astype(str).tolist()
    entries = []
    for subject_id in positive_subjects:
        center, _ = pilot_positive_center(subject_id, rng, epoch)
        entries.append({
            'sample_type': 'positive',
            'subject_id': subject_id,
            'center': center,
            'topology_relation': '',
        })
    for _ in range(topology_count):
        selected = None
        start = int(rng.integers(0, len(topology_pool)))
        for attempt in range(len(topology_pool)):
            subject_id = topology_pool[(start + attempt) % len(topology_pool)]
            support = pilot_unpack_mask(
                pilot_mask_records[subject_id]['sampling_support']
            )
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            region = pilot_load_region_map(subject_id)
            labels, relation_by_label = pilot_topology_candidates(support, region)
            for relation in [2, 1, 3]:
                relation_labels = [
                    label for label in labels
                    if relation_by_label[label] == relation
                ]
                if not relation_labels:
                    continue
                center, _ = pilot_zero_support_center(
                    exclusion, np.isin(region, relation_labels) & crop_mask, rng
                )
                if center is not None:
                    selected = {
                        'sample_type': 'topology_negative',
                        'subject_id': subject_id,
                        'center': center,
                        'topology_relation': PILOT_RELATION_NAME[relation],
                    }
                    break
            if selected is not None:
                break
        if selected is None:
            raise RuntimeError('Could not construct a topology-negative patch.')
        entries.append(selected)
    for _ in range(brain_count):
        selected = None
        start = int(rng.integers(0, len(brain_pool)))
        for attempt in range(len(brain_pool)):
            subject_id = brain_pool[(start + attempt) % len(brain_pool)]
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            center, _ = pilot_zero_support_center(exclusion, crop_mask, rng)
            if center is not None:
                selected = {
                    'sample_type': 'brain_negative',
                    'subject_id': subject_id,
                    'center': center,
                    'topology_relation': '',
                }
                break
        if selected is None:
            raise RuntimeError('Could not construct a brain-negative patch.')
        entries.append(selected)
    rng.shuffle(entries)
    return entries






graph_input_cache = {}
clinical_input_cache = {}


def pilot_load_graph_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in graph_input_cache:
        return graph_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: graph sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    territory_mean_local = np.asarray(
        fold0_graph_scaler['territory_mean'], dtype=np.float32
    )
    territory_std_local = np.asarray(
        fold0_graph_scaler['territory_std'], dtype=np.float32
    )
    vessel_indices = np.asarray(
        fold0_graph_scaler['vessel_source_indices'], dtype=np.int64
    )
    case_indices = np.asarray(
        fold0_graph_scaler['case_source_indices'], dtype=np.int64
    )
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_indices] & vessel_presence[:, None]
    )
    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean_local[None],
        territory_std_local[None],
    )
    vessel_scaled = masked_scale(
        vessel_source[:, vessel_indices],
        vessel_feature_valid,
        np.asarray(fold0_graph_scaler['vessel_mean'], dtype=np.float32)[None],
        np.asarray(fold0_graph_scaler['vessel_std'], dtype=np.float32)[None],
    )
    case_selected = case_source[case_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scale_mask_local = np.asarray(
        fold0_graph_scaler['case_scale_mask']
    ) > 0
    case_mean_local = np.asarray(
        fold0_graph_scaler['case_mean'], dtype=np.float32
    )
    case_std_local = np.asarray(
        fold0_graph_scaler['case_std'], dtype=np.float32
    )
    case_scaled[case_scale_mask_local] = np.clip(
        (
            case_scaled[case_scale_mask_local]
            - case_mean_local[case_scale_mask_local]
        ) / case_std_local[case_scale_mask_local],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean',
        'minimum_lvo_distance_mm', 'lvo_overlap_fraction',
    ]
    occlusion_graph_values = []
    occlusion_graph_valid = []
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = vessel_feature_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_graph_values.append(float(
                vessel_scaled[valid_nodes, feature_index].mean()
            ))
            occlusion_graph_valid.append(1.0)
        else:
            occlusion_graph_values.append(0.0)
            occlusion_graph_valid.append(0.0)
    output = {
        'territory_features': territory_scaled.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': vessel_scaled.astype(np.float32),
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence.astype(bool),
        'case_features': case_scaled.astype(np.float32),
        'cow_prior_valid': bool(
            case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
        ),
        'registration_confidence': float(
            case_source[
                SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
            ]
        ),
        'occlusion_graph_values': np.asarray(
            occlusion_graph_values, dtype=np.float32
        ),
        'occlusion_graph_valid': np.asarray(
            occlusion_graph_valid, dtype=np.float32
        ),
    }
    if not all(
        np.isfinite(value).all()
        for value in output.values()
        if np.asarray(value).dtype.kind in {'f', 'c'}
    ):
        raise RuntimeError(f'{subject_id}: scaled graph input is non-finite.')
    graph_input_cache[subject_id] = output
    return output


def pilot_load_clinical_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in clinical_input_cache:
        return clinical_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_clinical_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: clinical sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        names = decode_strings(data['feature_names'])
    scaler_names = decode_strings(fold0_clinical_scaler['feature_names'])
    if names != scaler_names or values.shape != (22,) or valid.shape != (22,):
        raise RuntimeError(f'{subject_id}: clinical schema changed.')
    imputed = values.copy()
    imputation = np.asarray(
        fold0_clinical_scaler['imputation'], dtype=np.float32
    )
    imputed[~valid] = imputation[~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(fold0_clinical_scaler['scale_mask']) > 0
    clinical_mean = np.asarray(fold0_clinical_scaler['mean'], dtype=np.float32)
    clinical_std = np.asarray(fold0_clinical_scaler['std'], dtype=np.float32)
    clinical_clip = float(
        np.asarray(fold0_clinical_scaler['clip_value']).reshape(-1)[0]
    )
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - clinical_mean[scale_mask])
        / clinical_std[scale_mask],
        -clinical_clip,
        clinical_clip,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    output = {
        'clinical_input': token,
        'clinical_available': np.asarray(valid.any(), dtype=bool),
    }
    clinical_input_cache[subject_id] = output
    return output


def pilot_model_input_from_patches(
    row,
    image_patch,
    region_patch,
    cow_patch,
    lvo_patch,
    lvo_present,
    apply_graph_mask=False,
    rng=None,
):
    graph = {
        key: np.asarray(value).copy()
        for key, value in pilot_load_graph_input(row).items()
    }
    clinical = pilot_load_clinical_input(row)
    if apply_graph_mask and rng is not None:
        mask = (
            rng.random(graph['territory_features'].shape) < 0.05
        ) & (graph['territory_feature_valid'] > 0)
        graph['territory_features'][mask] = 0.0
        graph['territory_feature_valid'][mask] = 0.0
    input_channels = EXPECTED_INPUT_CHANNELS
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch,
        lvo_patch,
    ]).astype(np.float32)
    confidence = np.asarray([
        float(vascular_patch[2].mean()),
        float(graph['cow_prior_valid']),
        float(np.clip(graph['registration_confidence'], 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_values = np.concatenate([
        np.asarray([float(lvo_present), float(lvo_patch.max())], dtype=np.float32),
        graph['occlusion_graph_values'],
    ]).astype(np.float32)
    occlusion_valid = np.concatenate([
        np.asarray([1.0, float(lvo_present)], dtype=np.float32),
        graph['occlusion_graph_valid'],
    ]).astype(np.float32)
    return {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': graph['territory_features'].astype(np.float32),
        'territory_feature_valid': graph[
            'territory_feature_valid'
        ].astype(np.float32),
        'territory_valid': graph['territory_valid'].astype(bool),
        'territory_reliability': graph['territory_reliability'].astype(np.float32),
        'vessel_features': graph['vessel_features'].astype(np.float32),
        'vessel_feature_valid': graph['vessel_feature_valid'].astype(np.float32),
        'vessel_valid': graph['vessel_valid'].astype(bool),
        'case_features': graph['case_features'].astype(np.float32),
        'atlas_enabled': np.asarray(
            as_bool(row['trial8_atlas_enabled']), dtype=bool
        ),
        'vascular_image': vascular_patch,
        'confidence_features': confidence,
        'occlusion_values': occlusion_values,
        'occlusion_valid': occlusion_valid,
        'clinical_input': clinical['clinical_input'].astype(np.float32),
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': clinical['clinical_available'],
    }


def pilot_intensity_augment(image, rng):
    output = np.asarray(image, dtype=np.float32).copy()
    intensity_channels = min(13, output.shape[0])
    gains = rng.uniform(0.92, 1.08, size=(intensity_channels, 1, 1, 1))
    offsets = rng.uniform(-0.02, 0.02, size=(intensity_channels, 1, 1, 1))
    output[:intensity_channels] = output[:intensity_channels] * gains + offsets
    if rng.random() < 0.50:
        noise_std = float(rng.uniform(0.0, 0.015))
        output[:intensity_channels] += rng.normal(
            0.0, noise_std, size=output[:intensity_channels].shape
        ).astype(np.float32)
    if rng.random() < 0.40:
        gamma = float(rng.uniform(0.90, 1.10))
        nonnegative_end = min(9, intensity_channels)
        output[:nonnegative_end] = np.power(
            np.clip(output[:nonnegative_end], 0.0, 1.0), gamma
        )
    output[:min(9, intensity_channels)] = np.clip(
        output[:min(9, intensity_channels)], 0.0, 1.0
    )
    if intensity_channels > 9:
        output[9:intensity_channels] = np.clip(
            output[9:intensity_channels], -1.0, 1.0
        )
    return output.astype(np.float32)


class Trial8Fold0PatchDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.counts = {}
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)
        self.counts = {
            name: sum(entry['sample_type'] == name for entry in self.schedule)
            for name in ['positive', 'topology_negative', 'brain_negative']
        }

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        with np.load(npz_path, allow_pickle=False) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            input_channels = decode_strings(data['input_channels'])
            target_binary = np.asarray(data['target_binary'], dtype=np.float32)
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
            lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
            lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            hard_weight = pilot_scalar(data, 'binary_loss_weight')
        if input_channels != EXPECTED_INPUT_CHANNELS:
            raise RuntimeError(f'{subject_id}: image channel order changed.')
        slices = pilot_patch_slices(target_binary.shape, entry['center'])
        image_patch = pilot_extract_patch(image, slices, (1, 2, 3))
        binary_patch = pilot_extract_patch(target_binary, slices, (0, 1, 2))
        soft_patch = pilot_extract_patch(target_soft, slices, (0, 1, 2))
        cow_patch = pilot_extract_patch(cow, slices, (0, 1, 2))
        lvo_patch = pilot_extract_patch(lvo, slices, (0, 1, 2))
        region_patch = pilot_extract_patch(
            pilot_load_region_map(subject_id), slices, (0, 1, 2)
        )
        if entry['sample_type'] == 'positive' and not bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: positive patch lost target support.')
        if entry['sample_type'] != 'positive' and bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: negative patch contains target support.')
        image_patch = pilot_intensity_augment(image_patch, rng)
        model_inputs = pilot_model_input_from_patches(
            row,
            image_patch,
            region_patch,
            cow_patch,
            lvo_patch,
            lvo_present,
            apply_graph_mask=True,
            rng=rng,
        )
        model_inputs.update({
            'target_binary': binary_patch[None].astype(np.float32),
            'target_soft': soft_patch[None].astype(np.float32),
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'supervision_mode': str(
                pilot_target_by_subject[subject_id]['supervision_mode']
            ),
        })
        return model_inputs


MODEL_INPUT_KEYS = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available',
]


def pilot_to_device(cpu_batch):
    output = {}
    for key, value in cpu_batch.items():
        if torch.is_tensor(value):
            output[key] = value.to(device, non_blocking=True)
        else:
            output[key] = value
    return output


def pilot_spatial_augment(batch):
    batch_size = int(batch['image'].shape[0])
    if float(torch.rand(1).item()) >= 0.60:
        batch['spatially_augmented_fraction'] = torch.tensor(
            0.0, device=batch['image'].device
        )
        return batch
    positive = [str(value) == 'positive' for value in batch['sample_type']]
    batch_device = batch['image'].device
    for _ in range(4):
        angles = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * math.radians(8.0)
        scales = 0.92 + torch.rand(batch_size, device=batch_device) * 0.16
        cosine = torch.cos(angles) * scales
        sine = torch.sin(angles) * scales
        theta = torch.zeros(
            batch_size, 3, 4, device=batch_device, dtype=torch.float32
        )
        theta[:, 0, 0] = cosine
        theta[:, 0, 1] = -sine
        theta[:, 1, 0] = sine
        theta[:, 1, 1] = cosine
        theta[:, 2, 2] = scales
        theta[:, 0, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        theta[:, 1, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        theta[:, 2, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.025
        grid = F.affine_grid(theta, batch['image'].shape, align_corners=False)

        transformed = dict(batch)
        transformed_image = torch.empty_like(batch['image'])
        transformed_image[:, :13] = F.grid_sample(
            batch['image'][:, :13], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed_image[:, 13:] = F.grid_sample(
            batch['image'][:, 13:], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['image'] = transformed_image
        transformed_vascular = torch.empty_like(batch['vascular_image'])
        continuous_indices = [0, 1, 3, 4]
        transformed_vascular[:, continuous_indices] = F.grid_sample(
            batch['vascular_image'][:, continuous_indices], grid,
            mode='bilinear', padding_mode='zeros', align_corners=False,
        )
        transformed_vascular[:, 2:3] = F.grid_sample(
            batch['vascular_image'][:, 2:3], grid,
            mode='nearest', padding_mode='zeros', align_corners=False,
        )
        transformed['vascular_image'] = transformed_vascular
        transformed['target_soft'] = F.grid_sample(
            batch['target_soft'], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed['target_binary'] = F.grid_sample(
            batch['target_binary'], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['region_map'] = F.grid_sample(
            batch['region_map'][:, None].float(), grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )[:, 0].long()
        positive_support_retained = all(
            (not is_positive) or bool(
                (transformed['target_soft'][batch_index] > 0).any().item()
            )
            for batch_index, is_positive in enumerate(positive)
        )
        if not positive_support_retained:
            continue
        transformed['confidence_features'] = batch[
            'confidence_features'
        ].clone()
        transformed['confidence_features'][:, 0] = transformed_vascular[
            :, 2
        ].float().mean(dim=(1, 2, 3))
        transformed['occlusion_values'] = batch['occlusion_values'].clone()
        transformed['occlusion_values'][:, 1] = transformed_vascular[
            :, 4
        ].float().amax(dim=(1, 2, 3))
        transformed['spatially_augmented_fraction'] = torch.tensor(
            1.0, device=batch_device
        )
        return transformed
    batch['spatially_augmented_fraction'] = torch.tensor(
        0.0, device=batch_device
    )
    return batch






def pilot_soft_dice_per_case(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_soft_focal_bce_per_case(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, raw.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_per_case(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error_floor = 1e-6
    return (
        (1.0 - tversky).clamp_min(error_floor).pow(gamma)
        - float(error_floor ** gamma)
    )


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    dice = pilot_soft_dice_per_case(logits, soft_target)
    focal_bce = pilot_soft_focal_bce_per_case(logits, soft_target)
    hard_tversky = pilot_focal_tversky_per_case(logits, binary_target)
    hard_contribution = 0.20 * hard_tversky * hard_weight.reshape(-1)
    total = 0.50 * dice + 0.30 * focal_bce + hard_contribution
    return total.mean(), {
        'soft_dice_loss': dice.mean(),
        'soft_focal_bce_loss': focal_bce.mean(),
        'hard_focal_tversky_loss': hard_tversky.mean(),
        'hard_contribution': hard_contribution.mean(),
    }


def pilot_node_loss(output, batch):
    node_logits = output['territory_node_logits'].float()
    targets = torch.zeros_like(node_logits)
    valid = (
        batch['territory_valid'].bool()
        & batch['atlas_enabled'][:, None].bool()
        & (batch['hard_weight'][:, None] > 0)
    )
    for batch_index in range(node_logits.shape[0]):
        labels = batch['region_map'][batch_index]
        lesion = batch['target_binary'][batch_index, 0] > 0.5
        if bool(lesion.any().item()):
            present_labels = torch.unique(labels[lesion])
            present_labels = present_labels[
                (present_labels >= 1) & (present_labels <= 30)
            ]
            if len(present_labels) > 0:
                targets[batch_index, present_labels.long() - 1] = 1.0
    if not bool(valid.any().item()):
        return node_logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        node_logits,
        targets,
        reduction='none',
        pos_weight=torch.full(
            (30,), 4.0, device=node_logits.device, dtype=node_logits.dtype
        ),
    )
    return raw[valid].mean()


def pilot_compute_training_loss(study_model, cpu_batch):
    batch = pilot_to_device(cpu_batch)
    batch = pilot_spatial_augment(batch)
    output = study_model(**{key: batch[key] for key in MODEL_INPUT_KEYS})
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    auxiliary_eligible = (batch['hard_weight'] > 0).float()
    half_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], size=output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'],
        size=output['auxiliary_quarter_logits'].shape[2:], mode='nearest',
    )
    half_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_half_logits'], half_binary
        ) * batch['hard_weight']
    )
    quarter_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_quarter_logits'], quarter_binary
        ) * batch['hard_weight']
    )
    denominator = auxiliary_eligible.sum().clamp_min(1.0)
    half_loss = (half_per_case * auxiliary_eligible).sum() / denominator
    quarter_loss = (quarter_per_case * auxiliary_eligible).sum() / denominator
    node_loss = pilot_node_loss(output, batch)
    total = main_loss + 0.15 * half_loss + 0.10 * quarter_loss + 0.02 * node_loss
    parts = {
        'total_loss': total,
        'main_loss': main_loss,
        **main_parts,
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
        'vascular_retained_fraction': output[
            'effective_vascular_enabled'
        ].float().mean(),
        'occlusion_retained_fraction': output[
            'effective_occlusion_enabled'
        ].float().mean(),
        'clinical_retained_fraction': output[
            'effective_clinical_enabled'
        ].float().mean(),
        'spatially_augmented_fraction': batch[
            'spatially_augmented_fraction'
        ],
    }
    return total, parts


def pilot_sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_extract_window(array, starts, spatial_axes):
    output_shape = list(array.shape)
    for axis, patch in zip(spatial_axes, PILOT_PATCH_SIZE):
        output_shape[axis] = patch
    output = np.zeros(output_shape, dtype=array.dtype)
    source = [slice(None)] * array.ndim
    destination = [slice(None)] * array.ndim
    source_spatial = []
    destination_spatial = []
    for axis, start, patch in zip(spatial_axes, starts, PILOT_PATCH_SIZE):
        stop = min(start + patch, array.shape[axis])
        copied = stop - start
        destination_start = max((patch - copied) // 2, 0)
        source[axis] = slice(start, stop)
        destination[axis] = slice(destination_start, destination_start + copied)
        source_spatial.append(slice(start, stop))
        destination_spatial.append(
            slice(destination_start, destination_start + copied)
        )
    output[tuple(destination)] = array[tuple(source)]
    return output, tuple(source_spatial), tuple(destination_spatial)


def pilot_gaussian_weight():
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in PILOT_PATCH_SIZE
    ]
    grids = np.meshgrid(*coordinates, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    weight = np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32)
    return np.maximum(weight, 0.05)


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


@torch.no_grad()
def pilot_full_volume_probability(study_model, row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: validation channel order changed.')
    region = pilot_load_region_map(subject_id)
    spatial_shape = image.shape[1:]
    probability_sum = np.zeros(spatial_shape, dtype=np.float32)
    weight_sum = np.zeros(spatial_shape, dtype=np.float32)
    window_records = []
    for x in pilot_sliding_starts(spatial_shape[0], PILOT_PATCH_SIZE[0]):
        for y in pilot_sliding_starts(spatial_shape[1], PILOT_PATCH_SIZE[1]):
            for z in pilot_sliding_starts(spatial_shape[2], PILOT_PATCH_SIZE[2]):
                window_records.append((x, y, z))
    study_model.eval()
    for start_index in range(0, len(window_records), 2):
        chunk = window_records[start_index:start_index + 2]
        numpy_inputs = []
        mappings = []
        for starts in chunk:
            image_patch, source_slices, destination_slices = pilot_extract_window(
                image, starts, (1, 2, 3)
            )
            region_patch, _, _ = pilot_extract_window(region, starts, (0, 1, 2))
            cow_patch, _, _ = pilot_extract_window(cow, starts, (0, 1, 2))
            lvo_patch, _, _ = pilot_extract_window(lvo, starts, (0, 1, 2))
            numpy_inputs.append(pilot_model_input_from_patches(
                row, image_patch, region_patch, cow_patch, lvo_patch,
                lvo_present, apply_graph_mask=False,
            ))
            mappings.append((source_slices, destination_slices))
        model_batch = {
            key: torch.from_numpy(np.stack([
                np.asarray(item[key]) for item in numpy_inputs
            ])).to(device, non_blocking=True)
            for key in MODEL_INPUT_KEYS
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            output = study_model(
                **model_batch,
                force_atlas_dropout=False,
                force_vascular_dropout=False,
                force_occlusion_dropout=False,
                force_clinical_dropout=False,
            )
        probabilities = torch.sigmoid(output['logits'].float())[
            :, 0
        ].cpu().numpy()
        for local_index, probability_patch in enumerate(probabilities):
            source_slices, destination_slices = mappings[local_index]
            patch_probability = probability_patch[destination_slices]
            patch_weight = PILOT_WINDOW_WEIGHT[destination_slices]
            probability_sum[source_slices] += patch_probability * patch_weight
            weight_sum[source_slices] += patch_weight
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    if not np.isfinite(probability).all() or bool((weight_sum <= 0).any()):
        raise RuntimeError(f'{subject_id}: invalid sliding-window output.')
    return probability.astype(np.float32), target_binary, spacing


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap_pairs = np.column_stack([
        predicted_labels[target & prediction],
        target_labels[target & prediction],
    ])
    overlap_pairs = overlap_pairs[
        (overlap_pairs[:, 0] > 0) & (overlap_pairs[:, 1] > 0)
    ]
    if len(overlap_pairs) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap_pairs, axis=0, return_counts=True)
    order = np.argsort(-counts)
    used_predicted = set()
    used_target = set()
    matches = 0
    for pair_index in order:
        predicted_id, target_id = [int(value) for value in pairs[pair_index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


PILOT_THRESHOLDS = np.round(np.arange(0.10, 0.81, 0.05), 2)


@torch.no_grad()
def pilot_validate(study_model, epoch):
    case_cache = []
    for case_index, row in enumerate(heldout_rows, start=1):
        probability, target, spacing = pilot_full_volume_probability(
            study_model, row
        )
        case_cache.append({
            'row': row,
            'probability': probability.astype(np.float16),
            'target': target,
            'spacing': spacing,
        })
        print(
            f'  Held-out inference {case_index:02d}/{len(heldout_rows)}: '
            f'{row["subject_id"]}'
        )
    threshold_rows = []
    case_threshold_rows = []
    for threshold in PILOT_THRESHOLDS:
        positive_dice = []
        global_intersection = 0
        global_prediction = 0
        global_target = 0
        for case in case_cache:
            row = case['row']
            prediction = case['probability'] >= threshold
            target = case['target']
            intersection = int((prediction & target).sum())
            predicted_voxels = int(prediction.sum())
            target_voxels = int(target.sum())
            dice = (
                (2.0 * intersection) / (predicted_voxels + target_voxels)
                if predicted_voxels + target_voxels > 0
                else 1.0
            )
            hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
            if hard_metric_eligible and target_voxels > 0:
                positive_dice.append(dice)
                global_intersection += intersection
                global_prediction += predicted_voxels
                global_target += target_voxels
            case_threshold_rows.append({
                'epoch': epoch,
                'subject_id': str(row['subject_id']),
                'threshold': float(threshold),
                'hard_metric_eligible': hard_metric_eligible,
                'target_voxels': target_voxels,
                'predicted_voxels': predicted_voxels,
                'intersection_voxels': intersection,
                'dice': float(dice),
            })
        if not positive_dice:
            raise RuntimeError('Fold-0 held-out set has no positive hard targets.')
        global_dice = (
            2.0 * global_intersection
            / max(global_prediction + global_target, 1)
        )
        threshold_rows.append({
            'threshold': float(threshold),
            'positive_mean_dice': float(np.mean(positive_dice)),
            'positive_median_dice': float(np.median(positive_dice)),
            'positive_global_dice': float(global_dice),
            'positive_subjects': len(positive_dice),
        })
    threshold_df = pd.DataFrame(threshold_rows).sort_values(
        ['positive_mean_dice', 'positive_global_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected_threshold = float(threshold_df.iloc[0]['threshold'])
    selected_case_rows = []
    lesion_tp = lesion_fp = lesion_fn = 0
    total_intersection = total_prediction = total_target = 0
    positive_dice = []
    precisions = []
    recalls = []
    volume_errors = []
    empty_fp_volumes = []
    bin_dice = {name: [] for name in POSITIVE_BINS}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        prediction = probability >= selected_threshold
        target = case['target']
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0
            else 1.0
        )
        precision = intersection / max(predicted_voxels, 1)
        recall = intersection / max(target_voxels, 1)
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        predicted_volume_ml = predicted_voxels * voxel_volume_ml
        target_volume_ml = target_voxels * voxel_volume_ml
        hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
        true_positive_lesions = false_positive_lesions = false_negative_lesions = 0
        if hard_metric_eligible:
            (
                true_positive_lesions,
                false_positive_lesions,
                false_negative_lesions,
            ) = pilot_lesion_counts(prediction, target)
            lesion_tp += true_positive_lesions
            lesion_fp += false_positive_lesions
            lesion_fn += false_negative_lesions
            volume_errors.append(abs(predicted_volume_ml - target_volume_ml))
            if target_voxels > 0:
                positive_dice.append(dice)
                precisions.append(precision)
                recalls.append(recall)
                total_intersection += intersection
                total_prediction += predicted_voxels
                total_target += target_voxels
                lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
                if lesion_bin in bin_dice:
                    bin_dice[lesion_bin].append(dice)
            else:
                empty_fp_volumes.append(predicted_volume_ml)
        selected_case_rows.append({
            'epoch': epoch,
            'subject_id': str(row['subject_id']),
            'threshold': selected_threshold,
            'supervision_mode': str(row['trial8_supervision_mode']),
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': hard_metric_eligible,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'precision': float(precision),
            'recall': float(recall),
            'target_volume_ml': target_volume_ml,
            'predicted_volume_ml': predicted_volume_ml,
            'absolute_volume_error_ml': abs(
                predicted_volume_ml - target_volume_ml
            ),
            'lesion_true_positive': true_positive_lesions,
            'lesion_false_positive': false_positive_lesions,
            'lesion_false_negative': false_negative_lesions,
        })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    result = {
        'threshold': selected_threshold,
        'positive_mean_dice': float(np.mean(positive_dice)),
        'positive_median_dice': float(np.median(positive_dice)),
        'positive_global_dice': float(
            2.0 * total_intersection / max(total_prediction + total_target, 1)
        ),
        'mean_voxel_precision': float(np.mean(precisions)),
        'mean_voxel_recall': float(np.mean(recalls)),
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'mean_absolute_volume_error_ml': float(np.mean(volume_errors)),
        'mean_empty_false_positive_volume_ml': float(
            np.mean(empty_fp_volumes) if empty_fp_volumes else 0.0
        ),
        'positive_subjects': len(positive_dice),
        'hard_metric_subjects': int(sum(
            as_bool(case['row']['trial8_hard_metric_eligible'])
            for case in case_cache
        )),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(threshold_df, epoch_root / 'threshold_summary.csv')
    atomic_write_csv(
        pd.DataFrame(case_threshold_rows), epoch_root / 'case_threshold_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(selected_case_rows), epoch_root / 'selected_case_metrics.csv'
    )
    atomic_write_json(result, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    return result






DIAGNOSTIC_VERSION = 'v2_2_fold0_failure_diagnostic_r1'
DIAGNOSTIC_CODE_REVISION = 'r1_branch_component_threshold_overfit_audit'
OFFICIAL_LESION_IOU_THRESHOLD = 0.20
BOOTSTRAP_REPLICATES = 5000
DIAGNOSTIC_THRESHOLDS = np.unique(np.concatenate([
    np.asarray([0.01, 0.02, 0.03, 0.04, 0.05, 0.075], dtype=np.float32),
    np.arange(0.10, 0.901, 0.025, dtype=np.float32),
])).astype(np.float32)
DIAGNOSTIC_MODES = [
    'full',
    'no_clinical',
    'no_occlusion',
    'no_vascular',
    'no_graph',
    'preserved_v1',
    'image_only',
]
SOURCE_CHECKPOINT_PATHS = {
    'v2': (
        PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
        / 'v1_fold0_v2_multimodal_ema_calibrated' / 'best_checkpoint.pt'
    ),
    'v2_1': (
        PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
        / 'v2_1_fold0_v2_multimodal_ema_calibrated' / 'best_checkpoint.pt'
    ),
}

DIAGNOSTIC_QA_ROOT = PILOT_ROOT / 'visual_qa'
DIAGNOSTIC_QA_ROOT.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_CONFIG_PATH = PILOT_ROOT / 'diagnostic_config.json'
DIAGNOSTIC_CHECKPOINT_AUDIT_PATH = PILOT_ROOT / 'source_checkpoint_audit.csv'
DIAGNOSTIC_MODE_SUMMARY_PATH = PILOT_ROOT / 'mode_summary.csv'
DIAGNOSTIC_CASE_METRICS_PATH = PILOT_ROOT / 'case_metrics.csv'
DIAGNOSTIC_THRESHOLD_PATH = PILOT_ROOT / 'threshold_sweep.csv'
DIAGNOSTIC_COMPONENT_PATH = PILOT_ROOT / 'target_component_audit.csv'
DIAGNOSTIC_HYSTERESIS_PATH = PILOT_ROOT / 'hysteresis_sweep.csv'
DIAGNOSTIC_GENERALIZATION_PATH = PILOT_ROOT / 'fit_heldout_generalization.csv'
DIAGNOSTIC_BOOTSTRAP_PATH = PILOT_ROOT / 'paired_bootstrap.csv'
DIAGNOSTIC_CORRELATION_PATH = PILOT_ROOT / 'quality_correlation_audit.csv'
DIAGNOSTIC_DECISION_PATH = PILOT_ROOT / 'diagnostic_decision_matrix.csv'
DIAGNOSTIC_SUMMARY_PATH = PILOT_ROOT / 'diagnostic_summary.json'
DIAGNOSTIC_GATE_PATH = PILOT_ROOT / 'diagnostic_gate.json'


def diagnostic_build_model():
    study_model = VascuFateNetV2(
        schema=graph_schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        graph_hidden=GRAPH_HIDDEN,
        graph_heads=GRAPH_HEADS,
        graph_layers=GRAPH_LAYERS,
        relation_bases=RELATION_BASES,
        graph_dropout=GRAPH_DROPOUT,
        atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
        film_bound=FILM_BOUND,
        vascular_channels=VASCULAR_CHANNELS,
        clinical_input_dim=CLINICAL_INPUT_DIM,
        clinical_embed_dim=CLINICAL_EMBED_DIM,
        occlusion_value_dim=OCCLUSION_VALUE_DIM,
        occlusion_embed_dim=OCCLUSION_EMBED_DIM,
        confidence_dim=CONFIDENCE_DIM,
        fusion_bound=FUSION_BOUND,
    ).to(device)
    if parameter_count(study_model) != EXPECTED_V2_PARAMETERS:
        raise RuntimeError('Diagnostic model parameter count changed.')
    return study_model


def diagnostic_checkpoint_state(checkpoint):
    if 'ema_model_state_dict' not in checkpoint:
        raise RuntimeError('Source checkpoint does not contain an EMA state.')
    return checkpoint['ema_model_state_dict']


def diagnostic_checkpoint_audit():
    missing = [
        str(path) for path in SOURCE_CHECKPOINT_PATHS.values()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            'Completed source checkpoint(s) are missing:\n' + '\n'.join(missing)
        )
    reference_model = diagnostic_build_model().cpu()
    reference_state = reference_model.state_dict()
    del reference_model
    rows = []
    checkpoints = {}
    state_key_sets = []
    for source_name, checkpoint_path in SOURCE_CHECKPOINT_PATHS.items():
        checkpoint = torch.load(
            checkpoint_path, map_location='cpu', weights_only=False
        )
        state = diagnostic_checkpoint_state(checkpoint)
        state_key_sets.append(set(state))
        missing_keys = sorted(set(reference_state) - set(state))
        extra_keys = sorted(set(state) - set(reference_state))
        shape_errors = sorted(
            name for name in set(reference_state) & set(state)
            if tuple(reference_state[name].shape) != tuple(state[name].shape)
        )
        fixed_threshold = float(checkpoint.get('best_threshold', np.nan))
        row = {
            'source_model': source_name,
            'checkpoint_path': str(checkpoint_path),
            'checkpoint_sha256': file_sha256(checkpoint_path),
            'pilot_version': str(checkpoint.get('pilot_version', '')),
            'code_revision': str(checkpoint.get('code_revision', '')),
            'best_epoch': int(checkpoint.get('best_epoch', -1)),
            'best_mean_dice': float(checkpoint.get('best_mean_dice', np.nan)),
            'fixed_threshold': fixed_threshold,
            'parameters': int(checkpoint.get('parameters', -1)),
            'state_keys': len(state),
            'missing_state_keys': len(missing_keys),
            'extra_state_keys': len(extra_keys),
            'shape_errors': len(shape_errors),
            'locked_validation_targets_read': bool(
                checkpoint.get('locked_validation_targets_read', True)
            ),
            'test_targets_read': bool(
                checkpoint.get('test_targets_read', True)
            ),
            'outcome_workbooks_read': bool(
                checkpoint.get('outcome_workbooks_read', True)
            ),
        }
        checks = {
            'parameter_count_matches': row['parameters'] == EXPECTED_V2_PARAMETERS,
            'state_keys_match': not missing_keys and not extra_keys,
            'state_shapes_match': not shape_errors,
            'fixed_threshold_is_valid': 0.0 < fixed_threshold < 1.0,
            'locked_validation_targets_not_read': not row[
                'locked_validation_targets_read'
            ],
            'test_targets_not_read': not row['test_targets_read'],
            'outcome_workbooks_not_read': not row['outcome_workbooks_read'],
        }
        row['checkpoint_contract_passed'] = bool(all(checks.values()))
        row['failed_checks'] = ';'.join(
            name for name, passed in checks.items() if not passed
        )
        rows.append(row)
        checkpoints[source_name] = checkpoint
    if state_key_sets[0] != state_key_sets[1]:
        raise RuntimeError('V2 and V2.1 EMA state keys differ.')
    audit = pd.DataFrame(rows)
    atomic_write_csv(audit, DIAGNOSTIC_CHECKPOINT_AUDIT_PATH)
    if not bool(audit['checkpoint_contract_passed'].all()):
        raise RuntimeError('Source checkpoint contract failed; see checkpoint audit.')
    return checkpoints, audit


def diagnostic_zero_graph_inputs(model_batch):
    zero_keys = [
        'territory_features', 'territory_feature_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'case_features',
    ]
    false_keys = ['territory_valid', 'vessel_valid', 'atlas_enabled']
    model_batch['region_map'] = torch.zeros_like(model_batch['region_map'])
    for key in zero_keys:
        model_batch[key] = torch.zeros_like(model_batch[key])
    for key in false_keys:
        model_batch[key] = torch.zeros_like(model_batch[key], dtype=torch.bool)


def diagnostic_mode_controls(model_batch, mode):
    if mode not in DIAGNOSTIC_MODES:
        raise ValueError(f'Unknown diagnostic mode: {mode}')
    no_graph = mode in {'no_graph', 'image_only'}
    if no_graph:
        diagnostic_zero_graph_inputs(model_batch)
    return {
        'force_atlas_dropout': bool(no_graph),
        'force_vascular_dropout': mode in {
            'no_vascular', 'preserved_v1', 'image_only'
        },
        'force_occlusion_dropout': mode in {
            'no_occlusion', 'preserved_v1', 'image_only'
        },
        'force_clinical_dropout': mode in {
            'no_clinical', 'preserved_v1', 'image_only'
        },
    }


@torch.no_grad()
def diagnostic_full_volume_probability(study_model, row, mode):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: diagnostic channel order changed.')
    region = pilot_load_region_map(subject_id)
    spatial_shape = image.shape[1:]
    probability_sum = np.zeros(spatial_shape, dtype=np.float32)
    weight_sum = np.zeros(spatial_shape, dtype=np.float32)
    gate_sums = {
        'context_gate_mean_abs': 0.0,
        'quarter_gate_mean_abs': 0.0,
        'eighth_gate_mean_abs': 0.0,
        'bottleneck_gate_mean_abs': 0.0,
    }
    gate_examples = 0
    window_records = [
        (x, y, z)
        for x in pilot_sliding_starts(spatial_shape[0], PILOT_PATCH_SIZE[0])
        for y in pilot_sliding_starts(spatial_shape[1], PILOT_PATCH_SIZE[1])
        for z in pilot_sliding_starts(spatial_shape[2], PILOT_PATCH_SIZE[2])
    ]
    study_model.eval()
    for start_index in range(0, len(window_records), 2):
        chunk = window_records[start_index:start_index + 2]
        numpy_inputs = []
        mappings = []
        for starts in chunk:
            image_patch, source_slices, destination_slices = pilot_extract_window(
                image, starts, (1, 2, 3)
            )
            region_patch, _, _ = pilot_extract_window(region, starts, (0, 1, 2))
            cow_patch, _, _ = pilot_extract_window(cow, starts, (0, 1, 2))
            lvo_patch, _, _ = pilot_extract_window(lvo, starts, (0, 1, 2))
            numpy_inputs.append(pilot_model_input_from_patches(
                row, image_patch, region_patch, cow_patch, lvo_patch,
                lvo_present, apply_graph_mask=False,
            ))
            mappings.append((source_slices, destination_slices))
        model_batch = {
            key: torch.from_numpy(np.stack([
                np.asarray(item[key]) for item in numpy_inputs
            ])).to(device, non_blocking=True)
            for key in MODEL_INPUT_KEYS
        }
        controls = diagnostic_mode_controls(model_batch, mode)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            output = study_model(**model_batch, **controls)
        probabilities = torch.sigmoid(output['logits'].float())[:, 0].cpu().numpy()
        gate_map = {
            'context_gate_mean_abs': 'context_fusion_gate',
            'quarter_gate_mean_abs': 'vascular_quarter_gate',
            'eighth_gate_mean_abs': 'vascular_eighth_gate',
            'bottleneck_gate_mean_abs': 'vascular_bottleneck_gate',
        }
        for result_name, output_name in gate_map.items():
            gate_sums[result_name] += float(
                output[output_name].float().abs().mean().cpu()
            ) * len(chunk)
        gate_examples += len(chunk)
        for local_index, probability_patch in enumerate(probabilities):
            source_slices, destination_slices = mappings[local_index]
            patch_probability = probability_patch[destination_slices]
            patch_weight = PILOT_WINDOW_WEIGHT[destination_slices]
            probability_sum[source_slices] += patch_probability * patch_weight
            weight_sum[source_slices] += patch_weight
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    if not np.isfinite(probability).all() or bool((weight_sum <= 0).any()):
        raise RuntimeError(f'{subject_id}: invalid diagnostic output.')
    gate_means = {
        name: value / max(gate_examples, 1)
        for name, value in gate_sums.items()
    }
    return (
        probability.astype(np.float32), target_binary, spacing,
        region.astype(np.uint8), gate_means,
    )


def diagnostic_dice(prediction, target):
    intersection = int((prediction & target).sum())
    denominator = int(prediction.sum()) + int(target.sum())
    return 2.0 * intersection / denominator if denominator else 1.0


def diagnostic_official_component_match(
    prediction, target, iou_threshold=OFFICIAL_LESION_IOU_THRESHOLD
):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    target_sizes = np.bincount(
        target_labels.reshape(-1), minlength=target_count + 1
    )
    overlap_mask = (predicted_labels > 0) & (target_labels > 0)
    matches = []
    if bool(overlap_mask.any()):
        predicted_overlap = predicted_labels[overlap_mask].astype(np.int64)
        target_overlap = target_labels[overlap_mask].astype(np.int64)
        pair_base = int(target_count) + 1
        pair_codes = predicted_overlap * pair_base + target_overlap
        unique_codes, overlap_counts = np.unique(pair_codes, return_counts=True)
        candidate_predicted = np.unique(unique_codes // pair_base)
        candidate_target = np.unique(unique_codes % pair_base)
        predicted_lookup = {
            int(value): index for index, value in enumerate(candidate_predicted)
        }
        target_lookup = {
            int(value): index for index, value in enumerate(candidate_target)
        }
        iou_matrix = np.zeros(
            (len(candidate_predicted), len(candidate_target)), dtype=np.float64
        )
        for code, overlap in zip(unique_codes, overlap_counts):
            predicted_id = int(code // pair_base)
            target_id = int(code % pair_base)
            union = (
                int(predicted_sizes[predicted_id])
                + int(target_sizes[target_id]) - int(overlap)
            )
            iou_matrix[
                predicted_lookup[predicted_id], target_lookup[target_id]
            ] = float(overlap) / max(union, 1)
        row_indices, column_indices = linear_sum_assignment(-iou_matrix)
        for row_index, column_index in zip(row_indices, column_indices):
            iou = float(iou_matrix[row_index, column_index])
            if iou >= iou_threshold:
                matches.append({
                    'predicted_component': int(candidate_predicted[row_index]),
                    'target_component': int(candidate_target[column_index]),
                    'iou': iou,
                })
    return {
        'matches': matches,
        'true_positive': len(matches),
        'false_positive': int(predicted_count - len(matches)),
        'false_negative': int(target_count - len(matches)),
        'predicted_count': int(predicted_count),
        'target_count': int(target_count),
        'predicted_labels': predicted_labels,
        'target_labels': target_labels,
        'predicted_sizes': predicted_sizes,
        'target_sizes': target_sizes,
    }


def diagnostic_surface_metrics(prediction, target, spacing):
    if not bool(prediction.any()) or not bool(target.any()):
        return {
            'surface_dice_2mm': np.nan,
            'surface_dice_3mm': np.nan,
            'hd95_mm': np.nan,
            'assd_mm': np.nan,
        }
    structure = ndimage.generate_binary_structure(3, 1)
    predicted_surface = prediction & ~ndimage.binary_erosion(
        prediction, structure=structure, border_value=0
    )
    target_surface = target & ~ndimage.binary_erosion(
        target, structure=structure, border_value=0
    )
    distance_to_target = ndimage.distance_transform_edt(
        ~target_surface, sampling=tuple(float(value) for value in spacing)
    )
    distance_to_prediction = ndimage.distance_transform_edt(
        ~predicted_surface, sampling=tuple(float(value) for value in spacing)
    )
    predicted_to_target = distance_to_target[predicted_surface]
    target_to_predicted = distance_to_prediction[target_surface]
    combined = np.concatenate([predicted_to_target, target_to_predicted])
    denominator = max(len(predicted_to_target) + len(target_to_predicted), 1)
    return {
        'surface_dice_2mm': float(
            ((predicted_to_target <= 2.0).sum()
             + (target_to_predicted <= 2.0).sum()) / denominator
        ),
        'surface_dice_3mm': float(
            ((predicted_to_target <= 3.0).sum()
             + (target_to_predicted <= 3.0).sum()) / denominator
        ),
        'hd95_mm': float(np.percentile(combined, 95)),
        'assd_mm': float(combined.mean()),
    }


def diagnostic_evaluate_cache(case_cache, threshold, include_surfaces=False):
    rows = []
    totals = {
        'intersection': 0, 'prediction': 0, 'target': 0,
        'lesion_tp': 0, 'lesion_fp': 0, 'lesion_fn': 0,
    }
    positive_dice = []
    bin_dice = {name: [] for name in POSITIVE_BINS}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = diagnostic_dice(prediction, target)
        precision = intersection / max(predicted_voxels, 1)
        recall = intersection / max(target_voxels, 1)
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        component = diagnostic_official_component_match(prediction, target)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
        if hard_eligible:
            totals['lesion_tp'] += component['true_positive']
            totals['lesion_fp'] += component['false_positive']
            totals['lesion_fn'] += component['false_negative']
            if target_voxels > 0:
                positive_dice.append(dice)
                if lesion_bin in bin_dice:
                    bin_dice[lesion_bin].append(dice)
                totals['intersection'] += intersection
                totals['prediction'] += predicted_voxels
                totals['target'] += target_voxels
        surface = (
            diagnostic_surface_metrics(prediction, target, case['spacing'])
            if include_surfaces and hard_eligible and target_voxels > 0
            else {
                'surface_dice_2mm': np.nan,
                'surface_dice_3mm': np.nan,
                'hd95_mm': np.nan,
                'assd_mm': np.nan,
            }
        )
        rows.append({
            'subject_id': str(row['subject_id']),
            'lesion_volume_bin': lesion_bin,
            'hard_metric_eligible': hard_eligible,
            'threshold': float(threshold),
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'intersection_voxels': intersection,
            'dice': float(dice),
            'voxel_precision': float(precision),
            'voxel_recall': float(recall),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml': predicted_voxels * voxel_volume_ml,
            'absolute_volume_error_ml': abs(
                predicted_voxels - target_voxels
            ) * voxel_volume_ml,
            'target_components': component['target_count'],
            'predicted_components': component['predicted_count'],
            'lesion_true_positive_iou20': component['true_positive'],
            'lesion_false_positive_iou20': component['false_positive'],
            'lesion_false_negative_iou20': component['false_negative'],
            **surface,
            **case['gate_means'],
        })
    lesion_precision = totals['lesion_tp'] / max(
        totals['lesion_tp'] + totals['lesion_fp'], 1
    )
    lesion_recall = totals['lesion_tp'] / max(
        totals['lesion_tp'] + totals['lesion_fn'], 1
    )
    lesion_f1 = 2.0 * lesion_precision * lesion_recall / max(
        lesion_precision + lesion_recall, 1e-12
    )
    finite_bin_means = [
        float(np.mean(values)) for values in bin_dice.values() if values
    ]
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': float(np.mean(positive_dice)),
        'positive_median_dice': float(np.median(positive_dice)),
        'positive_global_dice': float(
            2.0 * totals['intersection']
            / max(totals['prediction'] + totals['target'], 1)
        ),
        'macro_lesion_bin_dice': float(np.mean(finite_bin_means)),
        'official_lesion_precision_iou20': float(lesion_precision),
        'official_lesion_recall_iou20': float(lesion_recall),
        'official_lesion_f1_iou20': float(lesion_f1),
        'positive_subjects': len(positive_dice),
        'target_components': int(totals['lesion_tp'] + totals['lesion_fn']),
        'predicted_components': int(totals['lesion_tp'] + totals['lesion_fp']),
    }
    for gate_name in [
        'context_gate_mean_abs', 'quarter_gate_mean_abs',
        'eighth_gate_mean_abs', 'bottleneck_gate_mean_abs',
    ]:
        summary[gate_name] = float(np.mean([
            row[gate_name] for row in rows
        ]))
    for name, values in bin_dice.items():
        summary[f'mean_dice_{name}'] = (
            float(np.mean(values)) if values else np.nan
        )
    summary['pareto_score'] = float(
        0.60 * summary['positive_mean_dice']
        + 0.20 * summary['macro_lesion_bin_dice']
        + 0.20 * summary['official_lesion_f1_iou20']
    )
    return summary, rows


def diagnostic_threshold_sweep(case_cache, fixed_threshold=None):
    
    
    
    
    thresholds = np.asarray(DIAGNOSTIC_THRESHOLDS, dtype=np.float64)
    bins = np.concatenate(([-np.inf], thresholds, [np.inf]))
    accumulators = [{
        'positive_dice': [],
        'bin_dice': {name: [] for name in POSITIVE_BINS},
        'intersection': 0,
        'prediction': 0,
        'target': 0,
    } for _ in thresholds]
    case_dice_by_threshold = {float(value): {} for value in thresholds}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        target = case['target']
        all_histogram = np.histogram(probability, bins=bins)[0]
        target_histogram = np.histogram(probability[target], bins=bins)[0]
        predicted_counts = np.cumsum(all_histogram[::-1])[::-1][1:]
        intersection_counts = np.cumsum(target_histogram[::-1])[::-1][1:]
        target_voxels = int(target.sum())
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
        subject_id = str(row['subject_id'])
        for threshold_index, threshold in enumerate(thresholds):
            predicted_voxels = int(predicted_counts[threshold_index])
            intersection = int(intersection_counts[threshold_index])
            denominator = predicted_voxels + target_voxels
            dice = 2.0 * intersection / denominator if denominator else 1.0
            case_dice_by_threshold[float(threshold)][subject_id] = float(dice)
            if hard_eligible and target_voxels > 0:
                accumulator = accumulators[threshold_index]
                accumulator['positive_dice'].append(float(dice))
                if lesion_bin in accumulator['bin_dice']:
                    accumulator['bin_dice'][lesion_bin].append(float(dice))
                accumulator['intersection'] += intersection
                accumulator['prediction'] += predicted_voxels
                accumulator['target'] += target_voxels
    rows = []
    for threshold, accumulator in zip(thresholds, accumulators):
        bin_means = {
            name: float(np.mean(values)) if values else np.nan
            for name, values in accumulator['bin_dice'].items()
        }
        finite_bins = [value for value in bin_means.values() if math.isfinite(value)]
        rows.append({
            'threshold': float(threshold),
            'positive_mean_dice': float(np.mean(accumulator['positive_dice'])),
            'positive_median_dice': float(np.median(accumulator['positive_dice'])),
            'positive_global_dice': float(
                2.0 * accumulator['intersection']
                / max(accumulator['prediction'] + accumulator['target'], 1)
            ),
            'macro_lesion_bin_dice': float(np.mean(finite_bins)),
            'positive_subjects': len(accumulator['positive_dice']),
            'official_lesion_precision_iou20': np.nan,
            'official_lesion_recall_iou20': np.nan,
            'official_lesion_f1_iou20': np.nan,
            'pareto_score': np.nan,
            **{f'mean_dice_{name}': value for name, value in bin_means.items()},
        })
    table = pd.DataFrame(rows)
    calibrated = table.sort_values(
        ['positive_mean_dice', 'positive_global_dice', 'threshold'],
        ascending=[False, False, True],
    ).iloc[0].to_dict()
    top_dice_thresholds = table.nlargest(3, 'positive_mean_dice')[
        'threshold'
    ].tolist()
    anchor_thresholds = [0.10, 0.30, 0.50, 0.70]
    candidate_thresholds = set(top_dice_thresholds)
    candidate_thresholds.add(float(calibrated['threshold']))
    if fixed_threshold is not None:
        candidate_thresholds.add(float(fixed_threshold))
    for anchor in anchor_thresholds:
        candidate_thresholds.add(float(
            thresholds[np.argmin(np.abs(thresholds - anchor))]
        ))
    pareto_rows = []
    for threshold in sorted(candidate_thresholds):
        summary, _ = diagnostic_evaluate_cache(
            case_cache, threshold, include_surfaces=False
        )
        pareto_rows.append(summary)
        matching = np.isclose(table['threshold'].to_numpy(), threshold)
        for name in [
            'official_lesion_precision_iou20',
            'official_lesion_recall_iou20',
            'official_lesion_f1_iou20', 'target_components',
            'predicted_components', 'pareto_score',
        ]:
            table.loc[matching, name] = summary[name]
    pareto = pd.DataFrame(pareto_rows).sort_values(
        ['pareto_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).iloc[0].to_dict()
    oracle_rows = []
    for case in case_cache:
        subject_id = str(case['row']['subject_id'])
        candidates = [
            (float(threshold), case_dice_by_threshold[float(threshold)][subject_id])
            for threshold in DIAGNOSTIC_THRESHOLDS
        ]
        best_threshold, best_dice = sorted(
            candidates, key=lambda item: (-item[1], item[0])
        )[0]
        oracle_rows.append({
            'subject_id': subject_id,
            'oracle_threshold': best_threshold,
            'oracle_dice': float(best_dice),
        })
    return table, calibrated, pareto, pd.DataFrame(oracle_rows)


def diagnostic_hysteresis_prediction(
    probability, high_threshold, low_threshold, region_map=None,
    topology_gated=False,
):
    seeds = probability >= high_threshold
    low_mask = probability >= low_threshold
    if not bool(seeds.any()):
        return seeds
    if topology_gated and region_map is not None and bool((region_map > 0).any()):
        seed_labels = set(
            int(value) for value in np.unique(region_map[seeds]) if value > 0
        )
        allowed_labels = set(seed_labels)
        for edge_index in range(territory_edge_index_numpy.shape[1]):
            source = int(territory_edge_index_numpy[0, edge_index]) + 1
            destination = int(territory_edge_index_numpy[1, edge_index]) + 1
            if source in seed_labels or destination in seed_labels:
                allowed_labels.add(source)
                allowed_labels.add(destination)
        if allowed_labels:
            allowed = np.isin(region_map, sorted(allowed_labels))
            low_mask = low_mask & (allowed | seeds)
    return ndimage.binary_propagation(
        seeds, structure=np.ones((3, 3, 3), dtype=bool), mask=low_mask
    )


def diagnostic_hysteresis_sweep(case_cache, calibrated_threshold):
    high_values = sorted(set([
        float(calibrated_threshold), 0.30, 0.50, 0.70,
    ]))
    low_values = [0.01, 0.03, 0.05, 0.10, 0.20]
    rows = []
    for topology_gated in [False, True]:
        for high in high_values:
            for low in low_values:
                if low >= high:
                    continue
                dice_values = []
                tp = fp = fn = 0
                for case in case_cache:
                    prediction = diagnostic_hysteresis_prediction(
                        case['probability'].astype(np.float32), high, low,
                        case['region'], topology_gated=topology_gated,
                    )
                    target = case['target']
                    if as_bool(case['row']['trial8_hard_metric_eligible']) and bool(
                        target.any()
                    ):
                        dice_values.append(diagnostic_dice(prediction, target))
                    match = diagnostic_official_component_match(prediction, target)
                    tp += match['true_positive']
                    fp += match['false_positive']
                    fn += match['false_negative']
                precision = tp / max(tp + fp, 1)
                recall = tp / max(tp + fn, 1)
                rows.append({
                    'method': (
                        'topology_gated_hysteresis'
                        if topology_gated else 'spatial_hysteresis'
                    ),
                    'high_threshold': high,
                    'low_threshold': low,
                    'positive_mean_dice': float(np.mean(dice_values)),
                    'official_lesion_precision_iou20': float(precision),
                    'official_lesion_recall_iou20': float(recall),
                    'official_lesion_f1_iou20': float(
                        2.0 * precision * recall
                        / max(precision + recall, 1e-12)
                    ),
                })
    return pd.DataFrame(rows)


def diagnostic_target_component_rows(
    source_model, case_cache, selected_threshold
):
    rows = []
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    for case in case_cache:
        row = case['row']
        subject_id = str(row['subject_id'])
        probability = case['probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= selected_threshold
        selected_match = diagnostic_official_component_match(prediction, target)
        matched_by_target = {
            item['target_component']: item for item in selected_match['matches']
        }
        target_labels = selected_match['target_labels']
        target_count = selected_match['target_count']
        if bool(prediction.any()):
            distance_to_prediction = ndimage.distance_transform_edt(
                ~prediction,
                sampling=tuple(float(value) for value in case['spacing']),
            )
        else:
            distance_to_prediction = None
        oracle_by_target = {
            target_id: {'iou': 0.0, 'threshold': np.nan}
            for target_id in range(1, target_count + 1)
        }
        for threshold in DIAGNOSTIC_THRESHOLDS:
            candidate = probability >= float(threshold)
            candidate_match = diagnostic_official_component_match(
                candidate, target, iou_threshold=0.0
            )
            for item in candidate_match['matches']:
                target_id = item['target_component']
                if item['iou'] > oracle_by_target[target_id]['iou']:
                    oracle_by_target[target_id] = {
                        'iou': float(item['iou']),
                        'threshold': float(threshold),
                    }
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        for target_id in range(1, target_count + 1):
            mask = target_labels == target_id
            values = probability[mask]
            oracle = oracle_by_target[target_id]
            matched = matched_by_target.get(target_id)
            nearest_distance = (
                float(distance_to_prediction[mask].min())
                if distance_to_prediction is not None else np.nan
            )
            if matched is not None:
                failure_class = 'detected_iou20'
            elif oracle['iou'] >= OFFICIAL_LESION_IOU_THRESHOLD:
                failure_class = 'threshold_mismatch'
            elif float(values.max()) >= max(0.05, selected_threshold * 0.50):
                failure_class = 'topology_or_fragmentation'
            elif math.isfinite(nearest_distance) and nearest_distance <= 5.0:
                failure_class = 'near_miss_within_5mm'
            elif float(values.max()) >= 0.03:
                failure_class = 'weak_localization'
            else:
                failure_class = 'representation_failure'
            rows.append({
                'source_model': source_model,
                'subject_id': subject_id,
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'target_component': target_id,
                'component_voxels': int(mask.sum()),
                'component_volume_ml': float(mask.sum() * voxel_volume_ml),
                'selected_threshold': float(selected_threshold),
                'selected_iou': float(matched['iou']) if matched else 0.0,
                'oracle_iou': float(oracle['iou']),
                'oracle_threshold': oracle['threshold'],
                'probability_maximum': float(values.max()),
                'probability_mean': float(values.mean()),
                'probability_p95': float(np.percentile(values, 95)),
                'nearest_selected_prediction_mm': nearest_distance,
                'failure_class': failure_class,
            })
    return rows


def diagnostic_visual_qa(
    source_model, row, probability, target, threshold, output_path
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)[0]
    coordinates = np.argwhere(target)
    if len(coordinates):
        center = np.rint(coordinates.mean(axis=0)).astype(int)
    else:
        center = np.asarray(np.unravel_index(np.argmax(probability), probability.shape))
    prediction = probability >= threshold
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    plane_names = ['sagittal', 'coronal', 'axial']
    for axis, plane_name in enumerate(plane_names):
        index = int(center[axis])
        image_plane = np.take(image, index, axis=axis).T
        target_plane = np.take(target, index, axis=axis).T
        probability_plane = np.take(probability, index, axis=axis).T
        prediction_plane = np.take(prediction, index, axis=axis).T
        axes[0, axis].imshow(image_plane, cmap='gray', origin='lower')
        if bool(target_plane.any()):
            axes[0, axis].contour(
                target_plane, levels=[0.5], colors='red', linewidths=1.0
            )
        axes[0, axis].set_title(f'{plane_name} {index}: target red')
        axes[1, axis].imshow(
            probability_plane, cmap='magma', origin='lower', vmin=0, vmax=1
        )
        if bool(target_plane.any()):
            axes[1, axis].contour(
                target_plane, levels=[0.5], colors='cyan', linewidths=1.0
            )
        if bool(prediction_plane.any()):
            axes[1, axis].contour(
                prediction_plane, levels=[0.5], colors='lime', linewidths=0.8
            )
        axes[1, axis].set_title(
            f'probability; target cyan, prediction lime @ {threshold:.3f}'
        )
        for row_index in range(2):
            axes[row_index, axis].axis('off')
    fig.suptitle(
        f'{source_model} {subject_id} '
        f'({row["trial8_effective_lesion_volume_bin"]})'
    )
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def diagnostic_bin_matched_fit_rows():
    selected = []
    heldout_positive = fold0_heldout_df[
        fold0_heldout_df['trial8_hard_metric_eligible'].map(as_bool)
        & fold0_heldout_df['trial8_sampling_positive'].map(as_bool)
    ]
    for lesion_bin, count in heldout_positive[
        'trial8_effective_lesion_volume_bin'
    ].value_counts().items():
        pool = fold0_fit_df[
            fold0_fit_df['trial8_hard_metric_eligible'].map(as_bool)
            & fold0_fit_df['trial8_sampling_positive'].map(as_bool)
            & fold0_fit_df['trial8_effective_lesion_volume_bin'].eq(lesion_bin)
        ].copy()
        pool['_selection_key'] = pool['subject_id'].astype(str).map(
            lambda value: hashlib.sha256(
                f'{PILOT_SEED}|fit_match|{lesion_bin}|{value}'.encode('utf-8')
            ).hexdigest()
        )
        selected.extend(
            pool.sort_values('_selection_key').head(int(count)).drop(
                columns=['_selection_key']
            ).to_dict('records')
        )
    return sorted(selected, key=lambda row: str(row['subject_id']))


def diagnostic_bootstrap_difference(
    left_values, right_values, seed, replicates=BOOTSTRAP_REPLICATES
):
    left = np.asarray(left_values, dtype=np.float64)
    right = np.asarray(right_values, dtype=np.float64)
    if left.shape != right.shape or left.ndim != 1 or len(left) == 0:
        raise ValueError('Paired bootstrap inputs are invalid.')
    difference = left - right
    rng = np.random.default_rng(seed)
    sampled = np.empty(replicates, dtype=np.float64)
    for index in range(replicates):
        selection = rng.integers(0, len(difference), size=len(difference))
        sampled[index] = difference[selection].mean()
    return {
        'subjects': len(difference),
        'mean_difference': float(difference.mean()),
        'ci95_lower': float(np.percentile(sampled, 2.5)),
        'ci95_upper': float(np.percentile(sampled, 97.5)),
        'probability_difference_positive': float((sampled > 0).mean()),
    }


def diagnostic_quality_features(row):
    subject_id = str(row['subject_id'])
    graph = pilot_load_graph_input(row)
    clinical = pilot_load_clinical_input(row)
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        perfusion_valid_fraction = float(
            image[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')].mean()
        )
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    return {
        'subject_id': subject_id,
        'atlas_enabled': float(as_bool(row['trial8_atlas_enabled'])),
        'registration_confidence': float(graph['registration_confidence']),
        'valid_territory_nodes': int(np.asarray(graph['territory_valid']).sum()),
        'valid_vessel_nodes': int(np.asarray(graph['vessel_valid']).sum()),
        'perfusion_valid_fraction': perfusion_valid_fraction,
        'lvo_present': float(lvo_present),
        'clinical_missing_features': int(
            np.asarray(clinical['clinical_input'])[22:].sum()
        ),
    }


def run_trial8_v22_failure_diagnostic():
    checkpoints, checkpoint_audit = diagnostic_checkpoint_audit()
    checkpoint_hashes = {
        row['source_model']: row['checkpoint_sha256']
        for row in checkpoint_audit.to_dict('records')
    }
    if DIAGNOSTIC_GATE_PATH.exists():
        existing = json.loads(DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8'))
        if (
            existing.get('diagnostic_passed') is True
            and existing.get('source_checkpoint_sha256') == checkpoint_hashes
            and existing.get('code_revision') == DIAGNOSTIC_CODE_REVISION
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 FAILURE DIAGNOSTIC ALREADY COMPLETE')
            print('=' * 80)
            print(f'Diagnostic gate: {DIAGNOSTIC_GATE_PATH}')
            print('No source checkpoint or diagnostic output was rewritten.')
            return

    diagnostic_config = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': DIAGNOSTIC_VERSION,
        'code_revision': DIAGNOSTIC_CODE_REVISION,
        'source_checkpoint_paths': {
            name: str(path) for name, path in SOURCE_CHECKPOINT_PATHS.items()
        },
        'source_checkpoint_sha256': checkpoint_hashes,
        'modes': DIAGNOSTIC_MODES,
        'thresholds': DIAGNOSTIC_THRESHOLDS.tolist(),
        'official_lesion_iou_threshold': OFFICIAL_LESION_IOU_THRESHOLD,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'fit_subjects': len(fold0_fit_df),
        'heldout_subjects': len(fold0_heldout_df),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'training_performed': False,
        'model_training_authorized': False,
    }
    atomic_write_json(diagnostic_config, DIAGNOSTIC_CONFIG_PATH)

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 READ-ONLY FAILURE DIAGNOSTIC START')
    print('=' * 80)
    print(f'Device:                             {device}')
    print(f'Fold-0 fit / held-out:              {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
    print(f'Source models:                      {list(SOURCE_CHECKPOINT_PATHS)}')
    print(f'Ablation modes:                     {DIAGNOSTIC_MODES}')
    print(f'Threshold candidates:               {len(DIAGNOSTIC_THRESHOLDS)}')
    print('Official lesion match:              one-to-one IoU >= 0.20')
    print('Locked validation/test targets:     sealed')
    print('Training/optimizer steps:            0')

    all_mode_summaries = []
    all_case_rows = []
    all_threshold_rows = []
    all_component_rows = []
    all_hysteresis_rows = []
    heldout_full_thresholds = {}
    heldout_full_case_rows = {}
    all_loaded_models_frozen = True
    quality_rows = [
        diagnostic_quality_features(row) for row in heldout_rows
    ]
    quality_df = pd.DataFrame(quality_rows)

    for source_index, source_name in enumerate(['v2', 'v2_1']):
        checkpoint = checkpoints[source_name]
        study_model = diagnostic_build_model()
        study_model.load_state_dict(
            diagnostic_checkpoint_state(checkpoint), strict=True
        )
        study_model.eval().requires_grad_(False)
        all_loaded_models_frozen = all_loaded_models_frozen and not any(
            parameter.requires_grad for parameter in study_model.parameters()
        )
        fixed_threshold = float(checkpoint['best_threshold'])
        for mode_index, mode in enumerate(DIAGNOSTIC_MODES, start=1):
            print(
                f'\n{source_name} mode {mode_index}/{len(DIAGNOSTIC_MODES)}: '
                f'{mode}'
            )
            case_cache = []
            for case_index, row in enumerate(heldout_rows, start=1):
                probability, target, spacing, region, gate_means = (
                    diagnostic_full_volume_probability(study_model, row, mode)
                )
                case_cache.append({
                    'row': row,
                    'probability': probability.astype(np.float16),
                    'target': target,
                    'spacing': spacing,
                    'region': region,
                    'gate_means': gate_means,
                })
                print(
                    f'  {case_index:02d}/{len(heldout_rows)} '
                    f'{row["subject_id"]}'
                )
            threshold_df, calibrated, pareto, oracle_df = (
                diagnostic_threshold_sweep(case_cache, fixed_threshold)
            )
            threshold_df.insert(0, 'mode', mode)
            threshold_df.insert(0, 'source_model', source_name)
            all_threshold_rows.extend(threshold_df.to_dict('records'))

            evaluation_specs = [
                ('checkpoint_fixed', fixed_threshold, False),
                ('mode_recalibrated', float(calibrated['threshold']), mode == 'full'),
                ('pareto_recalibrated', float(pareto['threshold']), False),
            ]
            for policy, threshold, include_surfaces in evaluation_specs:
                summary, case_rows = diagnostic_evaluate_cache(
                    case_cache, threshold, include_surfaces=include_surfaces
                )
                summary.update({
                    'source_model': source_name,
                    'mode': mode,
                    'evaluation_policy': policy,
                    'checkpoint_fixed_threshold': fixed_threshold,
                })
                all_mode_summaries.append(summary)
                oracle_lookup = oracle_df.set_index('subject_id').to_dict('index')
                for case_row in case_rows:
                    case_row.update({
                        'source_model': source_name,
                        'mode': mode,
                        'evaluation_policy': policy,
                        'oracle_threshold': oracle_lookup[
                            case_row['subject_id']
                        ]['oracle_threshold'],
                        'oracle_dice': oracle_lookup[
                            case_row['subject_id']
                        ]['oracle_dice'],
                        'oracle_minus_selected_dice': (
                            oracle_lookup[case_row['subject_id']]['oracle_dice']
                            - case_row['dice']
                        ),
                    })
                all_case_rows.extend(case_rows)

            if mode == 'full':
                calibrated_threshold = float(calibrated['threshold'])
                heldout_full_thresholds[source_name] = calibrated_threshold
                full_rows = [
                    row for row in all_case_rows
                    if row['source_model'] == source_name
                    and row['mode'] == 'full'
                    and row['evaluation_policy'] == 'mode_recalibrated'
                ]
                heldout_full_case_rows[source_name] = full_rows
                component_rows = diagnostic_target_component_rows(
                    source_name, case_cache, calibrated_threshold
                )
                all_component_rows.extend(component_rows)
                hysteresis_df = diagnostic_hysteresis_sweep(
                    case_cache, calibrated_threshold
                )
                hysteresis_df.insert(0, 'source_model', source_name)
                all_hysteresis_rows.extend(hysteresis_df.to_dict('records'))
                for case in case_cache:
                    lesion_bin = str(
                        case['row']['trial8_effective_lesion_volume_bin']
                    )
                    if lesion_bin in {'tiny', 'small'} and bool(case['target'].any()):
                        qa_path = (
                            DIAGNOSTIC_QA_ROOT
                            / f'{source_name}_{case["row"]["subject_id"]}_qa.png'
                        )
                        diagnostic_visual_qa(
                            source_name, case['row'],
                            case['probability'].astype(np.float32),
                            case['target'], calibrated_threshold, qa_path,
                        )
            del case_cache
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        del study_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mode_summary_df = pd.DataFrame(all_mode_summaries)
    case_metrics_df = pd.DataFrame(all_case_rows)
    threshold_sweep_df = pd.DataFrame(all_threshold_rows)
    component_df = pd.DataFrame(all_component_rows)
    hysteresis_df = pd.DataFrame(all_hysteresis_rows)
    atomic_write_csv(mode_summary_df, DIAGNOSTIC_MODE_SUMMARY_PATH)
    atomic_write_csv(case_metrics_df, DIAGNOSTIC_CASE_METRICS_PATH)
    atomic_write_csv(threshold_sweep_df, DIAGNOSTIC_THRESHOLD_PATH)
    atomic_write_csv(component_df, DIAGNOSTIC_COMPONENT_PATH)
    atomic_write_csv(hysteresis_df, DIAGNOSTIC_HYSTERESIS_PATH)

    fit_subset_rows = diagnostic_bin_matched_fit_rows()
    expected_matched_fit_subjects = int((
        fold0_heldout_df['trial8_hard_metric_eligible'].map(as_bool)
        & fold0_heldout_df['trial8_sampling_positive'].map(as_bool)
    ).sum())
    if len(fit_subset_rows) != expected_matched_fit_subjects:
        raise RuntimeError(
            'Could not construct an exactly bin-matched fit subset: '
            f'{len(fit_subset_rows)} vs {expected_matched_fit_subjects}.'
        )
    fit_subjects = {str(row['subject_id']) for row in fit_subset_rows}
    heldout_subjects = {str(row['subject_id']) for row in heldout_rows}
    fit_bin_counts = pd.Series([
        str(row['trial8_effective_lesion_volume_bin'])
        for row in fit_subset_rows
    ]).value_counts().to_dict()
    heldout_bin_counts = fold0_heldout_df.loc[
        fold0_heldout_df['trial8_hard_metric_eligible'].map(as_bool)
        & fold0_heldout_df['trial8_sampling_positive'].map(as_bool),
        'trial8_effective_lesion_volume_bin',
    ].astype(str).value_counts().to_dict()
    if fit_subjects & heldout_subjects:
        raise RuntimeError('Bin-matched fit and held-out sets overlap.')
    generalization_rows = []
    for source_name in ['v2', 'v2_1']:
        checkpoint = checkpoints[source_name]
        study_model = diagnostic_build_model()
        study_model.load_state_dict(
            diagnostic_checkpoint_state(checkpoint), strict=True
        )
        study_model.eval().requires_grad_(False)
        all_loaded_models_frozen = all_loaded_models_frozen and not any(
            parameter.requires_grad for parameter in study_model.parameters()
        )
        fit_cache = []
        print(f'\n{source_name}: bin-matched fit-subset inference...')
        for case_index, row in enumerate(fit_subset_rows, start=1):
            probability, target, spacing, region, gate_means = (
                diagnostic_full_volume_probability(study_model, row, 'full')
            )
            fit_cache.append({
                'row': row,
                'probability': probability.astype(np.float16),
                'target': target,
                'spacing': spacing,
                'region': region,
                'gate_means': gate_means,
            })
            print(
                f'  {case_index:02d}/{len(fit_subset_rows)} '
                f'{row["subject_id"]}'
            )
        fit_summary, fit_cases = diagnostic_evaluate_cache(
            fit_cache, heldout_full_thresholds[source_name],
            include_surfaces=False,
        )
        heldout_cases = heldout_full_case_rows[source_name]
        heldout_positive = [
            row['dice'] for row in heldout_cases
            if row['hard_metric_eligible'] and row['target_voxels'] > 0
        ]
        for partition, rows, mean_dice in [
            ('bin_matched_fit', fit_cases, fit_summary['positive_mean_dice']),
            ('heldout', heldout_cases, float(np.mean(heldout_positive))),
        ]:
            generalization_rows.append({
                'source_model': source_name,
                'partition': partition,
                'subjects': len(rows),
                'threshold_from_heldout_full': heldout_full_thresholds[source_name],
                'positive_mean_dice': mean_dice,
            })
            for lesion_bin in POSITIVE_BINS:
                values = [
                    item['dice'] for item in rows
                    if item['lesion_volume_bin'] == lesion_bin
                    and item['hard_metric_eligible']
                    and item['target_voxels'] > 0
                ]
                generalization_rows[-1][f'mean_dice_{lesion_bin}'] = (
                    float(np.mean(values)) if values else np.nan
                )
        del fit_cache, study_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    generalization_df = pd.DataFrame(generalization_rows)
    atomic_write_csv(generalization_df, DIAGNOSTIC_GENERALIZATION_PATH)

    bootstrap_rows = []
    calibrated_cases = case_metrics_df[
        case_metrics_df['evaluation_policy'].eq('mode_recalibrated')
    ]
    for source_index, source_name in enumerate(['v2', 'v2_1']):
        for policy_index, evaluation_policy in enumerate([
            'checkpoint_fixed', 'mode_recalibrated'
        ]):
            source = case_metrics_df[
                case_metrics_df['source_model'].eq(source_name)
                & case_metrics_df['evaluation_policy'].eq(evaluation_policy)
            ]
            full = source[source['mode'].eq('full')].set_index('subject_id')
            for mode_index, mode in enumerate(DIAGNOSTIC_MODES[1:], start=1):
                ablated = source[source['mode'].eq(mode)].set_index('subject_id')
                common = sorted(set(full.index) & set(ablated.index))
                result = diagnostic_bootstrap_difference(
                    full.loc[common, 'dice'].to_numpy(),
                    ablated.loc[common, 'dice'].to_numpy(),
                    seed=(
                        PILOT_SEED + policy_index * 10_000
                        + source_index * 100 + mode_index
                    ),
                )
                bootstrap_rows.append({
                    'comparison': f'{source_name}:full_minus_{mode}',
                    'source_model': source_name,
                    'evaluation_policy': evaluation_policy,
                    'left_mode': 'full',
                    'right_mode': mode,
                    'subset': 'all_heldout',
                    **result,
                })
                small_common = [
                    subject_id for subject_id in common
                    if str(full.loc[subject_id, 'lesion_volume_bin']) in {
                        'tiny', 'small'
                    }
                ]
                if small_common:
                    result = diagnostic_bootstrap_difference(
                        full.loc[small_common, 'dice'].to_numpy(),
                        ablated.loc[small_common, 'dice'].to_numpy(),
                        seed=(
                            PILOT_SEED + 1000 + policy_index * 10_000
                            + source_index * 100 + mode_index
                        ),
                    )
                    bootstrap_rows.append({
                        'comparison': f'{source_name}:full_minus_{mode}',
                        'source_model': source_name,
                        'evaluation_policy': evaluation_policy,
                        'left_mode': 'full',
                        'right_mode': mode,
                        'subset': 'tiny_and_small',
                        **result,
                    })
    v2_full = calibrated_cases[
        calibrated_cases['source_model'].eq('v2')
        & calibrated_cases['mode'].eq('full')
    ].set_index('subject_id')
    v21_full = calibrated_cases[
        calibrated_cases['source_model'].eq('v2_1')
        & calibrated_cases['mode'].eq('full')
    ].set_index('subject_id')
    common = sorted(set(v2_full.index) & set(v21_full.index))
    bootstrap_rows.append({
        'comparison': 'v2_1_full_minus_v2_full',
        'source_model': 'cross_model',
        'evaluation_policy': 'mode_recalibrated',
        'left_mode': 'v2_1_full',
        'right_mode': 'v2_full',
        'subset': 'all_heldout',
        **diagnostic_bootstrap_difference(
            v21_full.loc[common, 'dice'].to_numpy(),
            v2_full.loc[common, 'dice'].to_numpy(),
            seed=PILOT_SEED + 9000,
        ),
    })
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    atomic_write_csv(bootstrap_df, DIAGNOSTIC_BOOTSTRAP_PATH)

    correlation_rows = []
    for source_name in ['v2', 'v2_1']:
        full = calibrated_cases[
            calibrated_cases['source_model'].eq(source_name)
            & calibrated_cases['mode'].eq('full')
        ].merge(quality_df, on='subject_id', how='left')
        candidate_features = [
            'target_volume_ml', 'target_components', 'atlas_enabled',
            'registration_confidence', 'valid_territory_nodes',
            'valid_vessel_nodes', 'perfusion_valid_fraction', 'lvo_present',
            'clinical_missing_features',
        ]
        for feature_name in candidate_features:
            for outcome_name in [
                'dice', 'absolute_volume_error_ml',
                'oracle_minus_selected_dice',
            ]:
                valid = (
                    np.isfinite(full[feature_name])
                    & np.isfinite(full[outcome_name])
                )
                if (
                    int(valid.sum()) >= 3
                    and full.loc[valid, feature_name].nunique() > 1
                    and full.loc[valid, outcome_name].nunique() > 1
                ):
                    rho, p_value = spearmanr(
                        full.loc[valid, feature_name],
                        full.loc[valid, outcome_name],
                    )
                else:
                    rho, p_value = np.nan, np.nan
                correlation_rows.append({
                    'source_model': source_name,
                    'feature': feature_name,
                    'outcome': outcome_name,
                    'subjects': int(valid.sum()),
                    'spearman_rho': (
                        float(rho) if math.isfinite(rho) else np.nan
                    ),
                    'two_sided_p_value_descriptive_only': (
                        float(p_value) if math.isfinite(p_value) else np.nan
                    ),
                    'confirmatory_inference': False,
                })
    correlation_df = pd.DataFrame(correlation_rows)
    atomic_write_csv(correlation_df, DIAGNOSTIC_CORRELATION_PATH)

    decision_rows = []
    for source_name in ['v2', 'v2_1']:
        full_summary = mode_summary_df[
            mode_summary_df['source_model'].eq(source_name)
            & mode_summary_df['mode'].eq('full')
            & mode_summary_df['evaluation_policy'].eq('mode_recalibrated')
        ].iloc[0]
        full_cases = calibrated_cases[
            calibrated_cases['source_model'].eq(source_name)
            & calibrated_cases['mode'].eq('full')
        ]
        component_source = component_df[
            component_df['source_model'].eq(source_name)
        ]
        oracle_gap = float(full_cases['oracle_minus_selected_dice'].mean())
        best_hysteresis = hysteresis_df[
            hysteresis_df['source_model'].eq(source_name)
        ].sort_values('positive_mean_dice', ascending=False).iloc[0]
        hysteresis_gain = float(
            best_hysteresis['positive_mean_dice']
            - full_summary['positive_mean_dice']
        )
        representation_failure_fraction = float(
            component_source['failure_class'].eq(
                'representation_failure'
            ).mean()
        ) if len(component_source) else np.nan
        fit_mean = float(generalization_df[
            generalization_df['source_model'].eq(source_name)
            & generalization_df['partition'].eq('bin_matched_fit')
        ]['positive_mean_dice'].iloc[0])
        heldout_mean = float(full_summary['positive_mean_dice'])
        overfit_gap = fit_mean - heldout_mean
        criteria = [
            (
                'threshold_or_postprocessing_limited',
                oracle_gap >= 0.03 or hysteresis_gain >= 0.015,
                max(oracle_gap, hysteresis_gain),
                'oracle gap >=0.03 or hysteresis gain >=0.015',
                'retain dense calibration and evaluate graph-guided hysteresis',
            ),
            (
                'representation_limited',
                representation_failure_fraction >= 0.30,
                representation_failure_fraction,
                'at least 30% of target components have maximum probability <0.03',
                'add a high-resolution lesion-candidate head and component loss',
            ),
            (
                'component_localization_limited',
                float(full_summary['official_lesion_recall_iou20']) < 0.30,
                float(full_summary['official_lesion_recall_iou20']),
                'official IoU20 lesion recall <0.30',
                'prioritize component-balanced sampling and directed vascular context',
            ),
            (
                'generalization_limited',
                overfit_gap >= 0.08,
                overfit_gap,
                'bin-matched fit minus held-out mean Dice >=0.08',
                'increase regularization and reduce conditioning capacity before expansion',
            ),
        ]
        for criterion, triggered, observed, rule, action in criteria:
            decision_rows.append({
                'source_model': source_name,
                'criterion': criterion,
                'triggered': bool(triggered),
                'observed_value': observed,
                'predeclared_rule': rule,
                'recommended_action_if_triggered': action,
            })
    for record in bootstrap_df[
        bootstrap_df['subset'].eq('all_heldout')
        & bootstrap_df['source_model'].isin(['v2', 'v2_1'])
        & bootstrap_df['evaluation_policy'].eq('checkpoint_fixed')
    ].to_dict('records'):
        if record['ci95_lower'] > 0.005:
            interpretation = 'branch_supported'
        elif record['ci95_upper'] < -0.005:
            interpretation = 'branch_harmful_or_misapplied'
        else:
            interpretation = 'inconclusive'
        decision_rows.append({
            'source_model': record['source_model'],
            'criterion': record['comparison'],
            'triggered': interpretation != 'inconclusive',
            'observed_value': record['mean_difference'],
            'predeclared_rule': (
                'paired bootstrap CI entirely above +0.005 supports branch; '
                'entirely below -0.005 indicates harm'
            ),
            'recommended_action_if_triggered': interpretation,
        })
    decision_rows.append({
        'source_model': 'policy',
        'criterion': 'post_treatment_routing',
        'triggered': False,
        'observed_value': np.nan,
        'predeclared_rule': (
            'primary prediction remains pre-treatment; post-treatment technique '
            'and reperfusion outcomes are excluded'
        ),
        'recommended_action_if_triggered': (
            'defer treatment-conditioned routing to a separately labeled '
            'retrospective secondary study'
        ),
    })
    decision_rows.append({
        'source_model': 'policy',
        'criterion': 'venous_topology_input',
        'triggered': False,
        'observed_value': np.nan,
        'predeclared_rule': (
            'the frozen sidecars contain arterial territory/vessel topology, '
            'not a verified venous graph'
        ),
        'recommended_action_if_triggered': (
            'do not label arterial features as venous; require a separately '
            'registered, target-free venous source and QA gate before use'
        ),
    })
    decision_df = pd.DataFrame(decision_rows)
    atomic_write_csv(decision_df, DIAGNOSTIC_DECISION_PATH)

    expected_mode_evaluations = (
        len(SOURCE_CHECKPOINT_PATHS) * len(DIAGNOSTIC_MODES) * 3
    )
    diagnostic_checks = {
        'both_source_checkpoints_verified': len(checkpoint_audit) == 2
        and bool(checkpoint_audit['checkpoint_contract_passed'].all()),
        'all_fourteen_model_mode_inferences_completed': (
            mode_summary_df[['source_model', 'mode']].drop_duplicates().shape[0]
            == len(SOURCE_CHECKPOINT_PATHS) * len(DIAGNOSTIC_MODES)
        ),
        'all_evaluation_policies_completed': len(mode_summary_df)
        == expected_mode_evaluations,
        'all_heldout_subjects_evaluated_per_mode': bool(
            case_metrics_df.groupby([
                'source_model', 'mode', 'evaluation_policy'
            ])['subject_id'].nunique().eq(len(heldout_rows)).all()
        ),
        'official_iou20_matching_used': math.isclose(
            OFFICIAL_LESION_IOU_THRESHOLD, 0.20
        ),
        'dense_low_threshold_grid_used': float(DIAGNOSTIC_THRESHOLDS.min())
        <= 0.01,
        'component_failure_audit_nonempty': len(component_df) > 0,
        'spatial_and_topology_hysteresis_tested': set(
            hysteresis_df['method']
        ) == {'spatial_hysteresis', 'topology_gated_hysteresis'},
        'fit_subset_is_disjoint_from_heldout': not bool(
            fit_subjects & heldout_subjects
        ),
        'fit_subset_count_matches_heldout_positive_count': (
            len(fit_subset_rows) == expected_matched_fit_subjects
        ),
        'fit_subset_lesion_bins_exactly_match_heldout': (
            fit_bin_counts == heldout_bin_counts
        ),
        'paired_bootstrap_completed': len(bootstrap_df) >= 13,
        'quality_correlations_completed': len(correlation_df) == 54,
        'decision_matrix_written': len(decision_df) > 0,
        'model_parameters_frozen_during_diagnostic': all_loaded_models_frozen,
        'no_optimizer_steps_performed': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'outcome_workbooks_not_read': True,
        'post_treatment_fields_not_used': True,
        'source_checkpoints_not_modified': all(
            file_sha256(path) == checkpoint_hashes[name]
            for name, path in SOURCE_CHECKPOINT_PATHS.items()
        ),
        'diagnostic_output_is_isolated': str(PILOT_ROOT).endswith(PILOT_VERSION),
        'model_training_remains_locked': True,
    }
    diagnostic_passed = bool(all(diagnostic_checks.values()))

    concise_full = mode_summary_df[
        mode_summary_df['mode'].eq('full')
        & mode_summary_df['evaluation_policy'].eq('mode_recalibrated')
    ][[
        'source_model', 'threshold', 'positive_mean_dice',
        'positive_median_dice', 'positive_global_dice',
        'macro_lesion_bin_dice', 'official_lesion_f1_iou20',
        'official_lesion_recall_iou20', 'mean_dice_tiny',
        'mean_dice_small', 'mean_dice_medium', 'mean_dice_large',
        'mean_dice_very_large',
    ]]
    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': DIAGNOSTIC_VERSION,
        'code_revision': DIAGNOSTIC_CODE_REVISION,
        'diagnostic_passed': diagnostic_passed,
        'checks': diagnostic_checks,
        'source_checkpoint_sha256': checkpoint_hashes,
        'full_model_results': concise_full.to_dict('records'),
        'official_lesion_iou_threshold': OFFICIAL_LESION_IOU_THRESHOLD,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'fit_subset_subjects': len(fit_subset_rows),
        'heldout_subjects': len(heldout_rows),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'training_performed': False,
        'study_trainable_parameters': 0,
        'model_training_authorized': False,
        'next_architecture_authorized': False,
        'decision_matrix_path': str(DIAGNOSTIC_DECISION_PATH),
    }
    atomic_write_json(summary, DIAGNOSTIC_SUMMARY_PATH)
    atomic_write_json({
        **summary,
        'diagnostic_output_root': str(PILOT_ROOT),
        'mode_summary_path': str(DIAGNOSTIC_MODE_SUMMARY_PATH),
        'case_metrics_path': str(DIAGNOSTIC_CASE_METRICS_PATH),
        'component_audit_path': str(DIAGNOSTIC_COMPONENT_PATH),
        'hysteresis_path': str(DIAGNOSTIC_HYSTERESIS_PATH),
        'bootstrap_path': str(DIAGNOSTIC_BOOTSTRAP_PATH),
        'correlation_path': str(DIAGNOSTIC_CORRELATION_PATH),
        'visual_qa_root': str(DIAGNOSTIC_QA_ROOT),
    }, DIAGNOSTIC_GATE_PATH)

    print('\nFull-model recalibrated results:')
    display(concise_full)
    print('\nPredeclared decision matrix:')
    display(decision_df)
    print('\nDiagnostic checks:')
    display(pd.DataFrame([
        {'check': name, 'passed': passed}
        for name, passed in diagnostic_checks.items()
    ]))
    if not diagnostic_passed:
        failed = [
            name for name, passed in diagnostic_checks.items() if not passed
        ]
        raise RuntimeError(
            'TRIAL8 V2.2 FAILURE DIAGNOSTIC FAILED. Do not revise or train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 FAILURE DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Source checkpoints:                 {len(SOURCE_CHECKPOINT_PATHS)}')
    print(f'Model/ablation combinations:        {len(SOURCE_CHECKPOINT_PATHS) * len(DIAGNOSTIC_MODES)}')
    print(f'Held-out subjects per combination:  {len(heldout_rows)}')
    print(f'Bin-matched fit subjects:           {len(fit_subset_rows)}')
    print(f'Official lesion matching IoU:       {OFFICIAL_LESION_IOU_THRESHOLD:.2f}')
    print(f'Bootstrap replicates:               {BOOTSTRAP_REPLICATES}')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Outcome/post-treatment data read:   False')
    print('Training/optimizer steps:           0')
    print('Study trainable parameters:         0 (diagnostic only)')
    print('Model training authorized:          False')
    print('Next architecture authorized:       False (review evidence first)')
    print(f'Diagnostic root:                    {PILOT_ROOT}')
    print(f'Decision matrix:                    {DIAGNOSTIC_DECISION_PATH}')
    print(f'Diagnostic gate:                    {DIAGNOSTIC_GATE_PATH}')
    print('\nNext: check the full output tables and tiny/small visual-QA panels.')


run_trial8_v22_failure_diagnostic()


In [ ]:
import sys
import os
import gc
import copy
import json
import math
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name]
        )
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')
scipy = ensure_package('scipy')
matplotlib = ensure_package('matplotlib')

import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

plt.switch_backend('Agg')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')





PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
FEATURE_VERSION = 'v1_atlas30_vessel13_target_free'
VERIFICATION_VERSION = 'v1_full_integrity_and_semantics'
BASELINE_MODEL_VERSION = 'trial8_vascufate_net_v1'
MODEL_VERSION = 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
AVAILABILITY_VERSION = 'v1_feature_availability_and_test_time_contract'
PREFLIGHT_VERSION = 'v2_2_diagnostic_model_reconstruction'
CODE_REVISION = 'r4_v2_2_read_only_failure_diagnostic'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_VERIFIED_MANIFEST_SHA256 = (
    '7c1a89b0e426f4e4e2d55e9139611e7f31d825e2224bbc5b05043c0fe88f4c5a'
)
EXPECTED_SAFE_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_SOURCE_SCHEMA_SHA256 = (
    '5b1bfc4005b3e33f164ad78bc2836075ecf74569ef674f18487b82a3002c7b9e'
)
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_BASELINE_PARAMETERS = 15_052_628

VERIFIED_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_verified.csv'
)
GRAPH_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / FEATURE_VERSION
)
VERIFICATION_ROOT = GRAPH_ROOT / 'verification' / VERIFICATION_VERSION
VERIFICATION_GATE_PATH = VERIFICATION_ROOT / 'graph_verification_gate.json'
SAFE_GRAPH_SCHEMA_PATH = (
    VERIFICATION_ROOT / 'graph_schema_verified_pickle_free.npz'
)

AVAILABILITY_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / AVAILABILITY_VERSION
)
AVAILABILITY_GATE_PATH = AVAILABILITY_ROOT / 'feature_availability_gate.json'
INPUT_CONTRACT_PATH = AVAILABILITY_ROOT / 'architecture_input_contract.json'

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
SCALER_NPZ_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.npz'
SCALER_JSON_PATH = OUTPUT_ROOT / 'train_only_graph_scaler.json'
ARCHITECTURE_JSON_PATH = OUTPUT_ROOT / 'architecture_config.json'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_model_construction_gate.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_model_construction_audit.csv'

SEED = 20260927
PATCH_SIZE = (128, 128, 64)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window',
    'ncct_broad_window',
    'cta',
    'cbf',
    'cbv',
    'mtt',
    'tmax',
    'relative_cbf',
    'relative_cbv',
    'delta_cbf',
    'delta_cbv',
    'delta_mtt',
    'delta_tmax',
    'perfusion_valid',
    'contralateral_pair_valid',
]
SCALER_EPSILON = 1e-6
SCALER_CLIP = 5.0
EXPECTED_TOTAL_PARAMETERS = 15_428_468
PARAMETER_BUDGET_MINIMUM = 15_300_000
PARAMETER_BUDGET_MAXIMUM = 15_650_000

EXPECTED_NODE_FEATURE_NAMES = [
    f'{channel}_{stat}'
    for channel in [
        'ncct_brain_window',
        'ncct_broad_window',
        'cta',
        'cbf',
        'cbv',
        'mtt',
        'tmax',
        'relative_cbf',
        'relative_cbv',
        'delta_cbf',
        'delta_cbv',
        'delta_mtt',
        'delta_tmax',
    ]
    for stat in ['mean', 'std', 'p10', 'p90']
] + [
    'log1p_node_voxels',
    'log1p_node_volume_ml',
    'node_brain_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'mtt_valid_fraction',
    'mtt_interpolation_support_mean',
    'cow_soft_union_mean',
    'cow_soft_union_max',
    'cow_label_overlap_fraction',
    'lvo_proximity_mean',
    'lvo_proximity_max',
    'lvo_proximity_gt_0p5_fraction',
]

SOURCE_VESSEL_FEATURE_NAMES = [
    'present',
    'volume_ml',
    'relative_centroid_x_mm',
    'relative_centroid_y_mm',
    'relative_centroid_z_mm',
    'extent_x_mm',
    'extent_y_mm',
    'extent_z_mm',
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'cbv_mean',
    'mtt_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]

SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'atlas_mode_syn',
    'atlas_mode_affine',
    'atlas_mode_disabled',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]




MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
]
CASE_FEATURES_TO_SCALE = {
    'registration_confidence',
    'vessel_presence_fraction',
    'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction',
    'log1p_brain_volume_ml',
}

FORBIDDEN_TARGET_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}


def assert_project_path_is_safe(path):
    path = Path(path).resolve()
    root = PROJECT_ROOT.resolve()
    if path != root and root not in path.parents:
        raise ValueError(f'Unsafe path outside project root: {path}')


assert_project_path_is_safe(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return to_builtin(value.tolist())
    if isinstance(value, np.generic):
        return to_builtin(value.item())
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def decode_strings(array):
    decoded = []
    for value in np.asarray(array).tolist():
        if isinstance(value, bytes):
            value = value.decode('utf-8')
        decoded.append(str(value))
    return decoded


def scalar_string(value):
    value = np.asarray(value).item()
    if isinstance(value, bytes):
        value = value.decode('utf-8')
    return str(value)


def set_determinism(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_determinism(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True






required_paths = [
    VERIFIED_MANIFEST_PATH,
    VERIFICATION_GATE_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    AVAILABILITY_GATE_PATH,
    INPUT_CONTRACT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 model-preflight prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

verified_manifest_sha256 = file_sha256(VERIFIED_MANIFEST_PATH)
safe_schema_sha256 = file_sha256(SAFE_GRAPH_SCHEMA_PATH)
if verified_manifest_sha256 != EXPECTED_VERIFIED_MANIFEST_SHA256:
    raise RuntimeError(
        'Verified graph manifest hash changed. '
        f'Observed {verified_manifest_sha256}; '
        f'expected {EXPECTED_VERIFIED_MANIFEST_SHA256}.'
    )
if safe_schema_sha256 != EXPECTED_SAFE_SCHEMA_SHA256:
    raise RuntimeError(
        'Pickle-free graph schema hash changed. '
        f'Observed {safe_schema_sha256}; expected {EXPECTED_SAFE_SCHEMA_SHA256}.'
    )

verification_gate = json.loads(
    VERIFICATION_GATE_PATH.read_text(encoding='utf-8')
)
availability_gate = json.loads(
    AVAILABILITY_GATE_PATH.read_text(encoding='utf-8')
)
input_contract = json.loads(INPUT_CONTRACT_PATH.read_text(encoding='utf-8'))
if verification_gate.get('verification_passed') is not True:
    raise RuntimeError('Graph verification gate is not passed.')
if verification_gate.get('model_training_authorized') is not True:
    raise RuntimeError('Graph verification does not authorize model training.')
if verification_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Graph verification did not remain target-free.')
if verification_gate.get('verified_manifest_sha256') != verified_manifest_sha256:
    raise RuntimeError('Verification gate manifest hash mismatch.')
gate_safe_schema_hash = verification_gate.get(
    'pickle_free_graph_schema_sha256',
    verification_gate.get('safe_graph_schema_sha256'),
)
if gate_safe_schema_hash is not None and gate_safe_schema_hash != safe_schema_sha256:
    raise RuntimeError('Verification gate safe-schema hash mismatch.')
gate_schema_hash = verification_gate.get('graph_schema_sha256')
if gate_schema_hash not in {
    None,
    EXPECTED_SOURCE_SCHEMA_SHA256,
    safe_schema_sha256,
}:
    raise RuntimeError('Verification gate source-schema hash mismatch.')

if availability_gate.get('feature_availability_audit_passed') is not True:
    raise RuntimeError('Vascular-fusion feature-availability gate is not passed.')
if availability_gate.get('architecture_implementation_authorized') is not True:
    raise RuntimeError('Feature audit does not authorize architecture construction.')
if availability_gate.get('model_training_authorized') is not False:
    raise RuntimeError('Feature audit must leave model training locked.')
if availability_gate.get('target_arrays_read') is not False:
    raise RuntimeError('Feature audit did not remain target-free.')
if availability_gate.get('outcome_workbooks_read') is not False:
    raise RuntimeError('Feature audit read outcome workbooks.')
if availability_gate.get('manifest_sha256') != EXPECTED_TARGET_POLICY_MANIFEST_SHA256:
    raise RuntimeError('Feature audit target-policy manifest hash mismatch.')
if availability_gate.get('clinical_token_input_dimension') != CLINICAL_INPUT_DIM:
    raise RuntimeError('Clinical-token input dimension changed.')
if input_contract.get('architecture_version') != MODEL_VERSION:
    raise RuntimeError('Architecture input-contract version mismatch.')
if input_contract.get('model_training_authorized') is not False:
    raise RuntimeError('Input contract must leave model training locked.')
if input_contract.get('anti_leakage', {}).get('target_arrays_read') is not False:
    raise RuntimeError('Input contract did not remain target-free.')

manifest_df = pd.read_csv(VERIFIED_MANIFEST_PATH)
required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_mode',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_graph_schema_path',
    'trial8_graph_schema_sha256',
    'trial8_graph_feature_verification_passed',
    'trial8_graph_training_scaler_policy',
    'trial8_model_training_authorized',
}
missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
if missing_columns:
    raise RuntimeError(f'Verified manifest missing columns: {missing_columns}')

manifest_checks = {
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'subjects_unique': manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS,
    'split_counts_match': manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'verification_passed_for_all': manifest_df[
        'trial8_graph_feature_verification_passed'
    ].map(as_bool).all(),
    'training_authorized_for_all': manifest_df[
        'trial8_model_training_authorized'
    ].map(as_bool).all(),
    'train_only_scaler_policy_frozen': (
        manifest_df['trial8_graph_training_scaler_policy']
        == 'fit_on_train_only_then_freeze'
    ).all(),
    'manifest_schema_paths_exist': manifest_df['trial8_graph_schema_path'].map(
        lambda value: Path(str(value)).exists()
    ).all(),
    'manifest_schema_hash_is_verified_source_or_safe': manifest_df[
        'trial8_graph_schema_sha256'
    ].isin({EXPECTED_SOURCE_SCHEMA_SHA256, safe_schema_sha256}).all(),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('Verified manifest checks failed: ' + ', '.join(failed))

with np.load(SAFE_GRAPH_SCHEMA_PATH, allow_pickle=False) as schema:
    graph_schema = {name: np.asarray(schema[name]) for name in schema.files}

if any(array.dtype.kind == 'O' for array in graph_schema.values()):
    raise RuntimeError('Verified graph schema still contains object arrays.')
required_graph_schema_arrays = {
    'node_feature_names',
    'vessel_feature_names',
    'case_feature_names',
    'territory_edge_index',
    'territory_edge_type',
    'territory_edge_weight',
    'territory_relation_names',
    'vessel_edge_index',
    'vessel_edge_type',
    'vessel_edge_weight',
    'vessel_relation_names',
    'cross_edge_index',
    'cross_edge_type',
    'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(
    required_graph_schema_arrays - set(graph_schema)
)
if missing_schema_arrays:
    raise RuntimeError(
        'Verified graph schema missing arrays: '
        + ', '.join(missing_schema_arrays)
    )
if decode_strings(graph_schema['node_feature_names']) != EXPECTED_NODE_FEATURE_NAMES:
    raise RuntimeError('Territory feature schema changed.')
if decode_strings(graph_schema['vessel_feature_names']) != SOURCE_VESSEL_FEATURE_NAMES:
    raise RuntimeError('Vessel feature schema changed.')
if decode_strings(graph_schema['case_feature_names']) != SOURCE_CASE_FEATURE_NAMES:
    raise RuntimeError('Case feature schema changed.')

print(f'Trial8 v2 model preflight revision: {CODE_REVISION}')
print('Trial8 verified graph-data gate passed.')
print(f'Verified subjects: {len(manifest_df)}')
print(f'Split counts: {manifest_df["dev_split"].value_counts().to_dict()}')
print(f'Verified manifest SHA-256: {verified_manifest_sha256}')
print(f'Pickle-free schema SHA-256: {safe_schema_sha256}')
print('Vascular-fusion availability gate: passed')
print('Target arrays read: False')
print('Outcome workbooks read: False')
print('Study trainable parameters before model construction: 0')






train_df = manifest_df[manifest_df['dev_split'] == 'train'].copy()
if len(train_df) != EXPECTED_SPLITS['train']:
    raise RuntimeError('Training split subject count changed.')

territory_accumulators = [[] for _ in EXPECTED_NODE_FEATURE_NAMES]
vessel_source_indices = [
    SOURCE_VESSEL_FEATURE_NAMES.index(name)
    for name in MODEL_VESSEL_FEATURE_NAMES
]
case_source_indices = [
    SOURCE_CASE_FEATURE_NAMES.index(name) for name in MODEL_CASE_FEATURE_NAMES
]
vessel_accumulators = [[] for _ in MODEL_VESSEL_FEATURE_NAMES]
case_accumulators = [[] for _ in MODEL_CASE_FEATURE_NAMES]
scaler_fit_subjects = []

for row in train_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'Missing training sidecar: {sidecar_path}')
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: training sidecar hash mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: target array found in sidecar.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_features = np.asarray(data['case_features'], dtype=np.float32)

    for feature_index in range(len(EXPECTED_NODE_FEATURE_NAMES)):
        valid = node_valid & node_feature_valid[:, feature_index]
        territory_accumulators[feature_index].extend(
            node_features[valid, feature_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(vessel_source_indices):
        valid = vessel_presence & vessel_feature_valid[:, source_index]
        vessel_accumulators[model_index].extend(
            vessel_features[valid, source_index].astype(float).tolist()
        )
    for model_index, source_index in enumerate(case_source_indices):
        case_accumulators[model_index].append(float(case_features[source_index]))
    scaler_fit_subjects.append(subject_id)


def fit_scaler(accumulators, feature_names, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(len(feature_names), dtype=bool)
    scale_mask = np.asarray(scale_mask, dtype=bool)
    means = np.zeros(len(feature_names), dtype=np.float32)
    stds = np.ones(len(feature_names), dtype=np.float32)
    counts = np.zeros(len(feature_names), dtype=np.int64)
    raw_stds = np.zeros(len(feature_names), dtype=np.float32)
    constants = np.zeros(len(feature_names), dtype=np.uint8)

    for index, values in enumerate(accumulators):
        array = np.asarray(values, dtype=np.float64)
        if array.size == 0 or not np.isfinite(array).all():
            raise RuntimeError(
                f'Invalid train-only scaler support for {feature_names[index]}.'
            )
        counts[index] = array.size
        raw_std = float(array.std())
        raw_stds[index] = raw_std
        constants[index] = int(raw_std < SCALER_EPSILON)
        if scale_mask[index]:
            means[index] = float(array.mean())
            stds[index] = max(raw_std, SCALER_EPSILON)
        else:
            means[index] = 0.0
            stds[index] = 1.0
    return means, stds, counts, raw_stds, constants


territory_mean, territory_std, territory_count, territory_raw_std, territory_constant = (
    fit_scaler(territory_accumulators, EXPECTED_NODE_FEATURE_NAMES)
)
vessel_mean, vessel_std, vessel_count, vessel_raw_std, vessel_constant = fit_scaler(
    vessel_accumulators, MODEL_VESSEL_FEATURE_NAMES
)
case_scale_mask = np.asarray(
    [name in CASE_FEATURES_TO_SCALE for name in MODEL_CASE_FEATURE_NAMES],
    dtype=bool,
)
case_mean, case_std, case_count, case_raw_std, case_constant = fit_scaler(
    case_accumulators,
    MODEL_CASE_FEATURE_NAMES,
    scale_mask=case_scale_mask,
)

scaler_checks = {
    'fit_subjects_are_exactly_train_split': set(scaler_fit_subjects)
    == set(train_df['subject_id'].astype(str)),
    'fit_subject_count_is_102': len(scaler_fit_subjects) == 102,
    'territory_features_all_supported': bool((territory_count > 0).all()),
    'vessel_features_all_supported': bool((vessel_count > 0).all()),
    'case_features_all_supported': bool((case_count > 0).all()),
    'territory_selected_features_nonconstant': not territory_constant.any(),
    'vessel_present_feature_removed': 'present' not in MODEL_VESSEL_FEATURE_NAMES,
    'vessel_selected_features_nonconstant': not vessel_constant.any(),
    'transform_mode_one_hots_removed': not any(
        name.startswith('atlas_mode_') for name in MODEL_CASE_FEATURE_NAMES
    ),
    'scaled_case_features_nonconstant': not case_constant[case_scale_mask].any(),
    'all_scaler_values_finite': all(
        np.isfinite(array).all()
        for array in [
            territory_mean,
            territory_std,
            vessel_mean,
            vessel_std,
            case_mean,
            case_std,
        ]
    ),
    'all_scaler_stds_positive': bool(
        (territory_std > 0).all()
        and (vessel_std > 0).all()
        and (case_std > 0).all()
    ),
    'validation_and_test_not_read_for_scaler': True,
    'target_arrays_not_read_for_scaler': True,
}
if not all(scaler_checks.values()):
    failed = [name for name, passed in scaler_checks.items() if not passed]
    raise RuntimeError('Train-only scaler checks failed: ' + ', '.join(failed))

scaler_fit_subject_hash = hashlib.sha256(
    '\n'.join(sorted(scaler_fit_subjects)).encode('utf-8')
).hexdigest()

atomic_write_npz(
    SCALER_NPZ_PATH,
    territory_feature_names=np.asarray(EXPECTED_NODE_FEATURE_NAMES),
    territory_mean=territory_mean,
    territory_std=territory_std,
    territory_count=territory_count,
    vessel_feature_names=np.asarray(MODEL_VESSEL_FEATURE_NAMES),
    vessel_source_indices=np.asarray(vessel_source_indices, dtype=np.int16),
    vessel_mean=vessel_mean,
    vessel_std=vessel_std,
    vessel_count=vessel_count,
    case_feature_names=np.asarray(MODEL_CASE_FEATURE_NAMES),
    case_source_indices=np.asarray(case_source_indices, dtype=np.int16),
    case_scale_mask=case_scale_mask.astype(np.uint8),
    case_mean=case_mean,
    case_std=case_std,
    case_count=case_count,
    clip_value=np.asarray(SCALER_CLIP, dtype=np.float32),
    fit_split=np.asarray('train'),
    fit_subject_count=np.asarray(len(scaler_fit_subjects), dtype=np.int16),
    fit_subject_hash=np.asarray(scaler_fit_subject_hash),
    target_arrays_read=np.asarray(False, dtype=np.uint8),
)
scaler_sha256 = file_sha256(SCALER_NPZ_PATH)

scaler_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'preflight_version': PREFLIGHT_VERSION,
    'fit_split': 'train',
    'fit_subject_count': len(scaler_fit_subjects),
    'fit_subject_hash': scaler_fit_subject_hash,
    'verified_manifest_sha256': verified_manifest_sha256,
    'scaler_npz_path': str(SCALER_NPZ_PATH),
    'scaler_npz_sha256': scaler_sha256,
    'territory_features': EXPECTED_NODE_FEATURE_NAMES,
    'vessel_features': MODEL_VESSEL_FEATURE_NAMES,
    'removed_vessel_features': ['present'],
    'case_features': MODEL_CASE_FEATURE_NAMES,
    'removed_case_features': [
        'atlas_mode_syn',
        'atlas_mode_affine',
        'atlas_mode_disabled',
    ],
    'case_features_scaled': sorted(CASE_FEATURES_TO_SCALE),
    'clip_value': SCALER_CLIP,
    'validation_subjects_read': 0,
    'test_subjects_read': 0,
    'target_arrays_read': False,
    'study_trainable_parameters': 0,
}
atomic_write_json(scaler_record, SCALER_JSON_PATH)

print('\nTrain-only scaler checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in scaler_checks.items()
]))
print(f'Train-only scaler SHA-256: {scaler_sha256}')






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]

        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)

        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self,
        hidden_dim,
        heads,
        num_relations,
        num_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(
            torch.empty(heads, self.head_dim)
        )
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self,
        vessel_nodes,
        territory_nodes,
        vessel_valid,
        territory_valid,
        edge_index,
        edge_type,
        edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size,
            territory_count,
            dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(dim=1, keepdim=True).clamp_min(
                1e-6
            )
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)

        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(
            has_message[..., None], candidate, territory_nodes
        )
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self,
        territory_input_dim,
        vessel_input_dim,
        case_input_dim,
        schema,
        hidden_dim=64,
        heads=2,
        layers=2,
        relation_bases=3,
        dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        territory_relations = len(decode_strings(schema['territory_relation_names']))
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                territory_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim,
                heads,
                vessel_relations,
                relation_bases,
                dropout,
            )
            for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim,
            heads,
            cross_relations,
            relation_bases,
            dropout,
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)

        self.register_buffer(
            'territory_edge_index',
            torch.as_tensor(schema['territory_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_type',
            torch.as_tensor(schema['territory_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'territory_edge_weight',
            torch.as_tensor(schema['territory_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'vessel_edge_index',
            torch.as_tensor(schema['vessel_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_type',
            torch.as_tensor(schema['vessel_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'vessel_edge_weight',
            torch.as_tensor(schema['vessel_edge_weight'], dtype=torch.float32),
        )
        self.register_buffer(
            'cross_edge_index',
            torch.as_tensor(schema['cross_edge_index'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_type',
            torch.as_tensor(schema['cross_edge_type'], dtype=torch.long),
        )
        self.register_buffer(
            'cross_edge_weight',
            torch.as_tensor(schema['cross_edge_weight'], dtype=torch.float32),
        )

    def forward(
        self,
        territory_input,
        vessel_input,
        case_input,
        territory_valid,
        vessel_valid,
        territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory,
                territory_valid,
                self.territory_edge_index,
                self.territory_edge_type,
                self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel,
                vessel_valid,
                self.vessel_edge_index,
                self.vessel_edge_type,
                self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel,
            territory,
            vessel_valid,
            territory_valid,
            self.cross_edge_index,
            self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(territory_reliability.dtype),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits






def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False,
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(
            in_channels, out_channels, kernel_size=2, stride=2
        )
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2),
            nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype),
            size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    

    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(
                math.atanh(float(fraction_of_bound))
            )


class AsymmetricVascularStream3D(nn.Module):
    

    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        pooled = v5.mean(dim=(2, 3, 4))
        return v3, v4, v5, pooled


class SpatialConfidenceFusion3D(nn.Module):
    

    def __init__(
        self,
        vascular_channels,
        image_channels,
        confidence_dim=4,
        bound=0.1,
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, kernel_size=1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound
        )

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature,
                size=image_feature.shape[2:],
                mode='trilinear',
                align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.value_dim = int(value_dim)
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        safe_values = values * valid
        return self.encoder(torch.cat([safe_values, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    

    def __init__(
        self,
        vascular_dim,
        occlusion_dim,
        clinical_dim,
        output_dim=64,
        confidence_dim=4,
        bound=0.1,
        dropout=0.10,
    ):
        super().__init__()
        source_dim = vascular_dim + occlusion_dim + clinical_dim
        self.source = nn.Sequential(
            nn.Linear(source_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self,
        graph_context,
        vascular_context,
        occlusion_context,
        clinical_context,
        confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context,
            occlusion_context,
            clinical_context,
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self,
        schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        channels=(32, 64, 128, 224, 320),
        graph_hidden=64,
        graph_heads=2,
        graph_layers=2,
        relation_bases=3,
        graph_dropout=0.1,
        atlas_dropout=0.2,
        film_bound=0.1,
        vascular_channels=(8, 16, 32, 48),
        clinical_input_dim=44,
        clinical_embed_dim=32,
        occlusion_value_dim=7,
        occlusion_embed_dim=32,
        confidence_dim=4,
        fusion_bound=0.1,
        vascular_dropout=0.15,
        clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        territory_input_dim = territory_feature_dim * 2 + 1
        vessel_input_dim = vessel_feature_dim * 2
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_input_dim,
            vessel_input_dim,
            case_feature_dim,
            schema,
            hidden_dim=graph_hidden,
            heads=graph_heads,
            layers=graph_layers,
            relation_bases=relation_bases,
            dropout=graph_dropout,
        )

        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)

        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        vc1, vc2, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vascular_dim=vc4,
            occlusion_dim=occlusion_embed_dim,
            clinical_dim=clinical_embed_dim,
            output_dim=graph_hidden,
            confidence_dim=confidence_dim,
            bound=fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, kernel_size=1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, kernel_size=1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, kernel_size=1)

    def _effective_atlas(
        self,
        atlas_enabled,
        force_atlas_dropout=None,
    ):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        modules = [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]
        for module in modules:
            module.activate_for_gradient_test(0.5)

    def forward(
        self,
        image,
        region_map,
        territory_features,
        territory_feature_valid,
        territory_valid,
        territory_reliability,
        vessel_features,
        vessel_feature_valid,
        vessel_valid,
        case_features,
        atlas_enabled,
        vascular_image,
        confidence_features,
        occlusion_values,
        occlusion_valid,
        clinical_input,
        vascular_available,
        occlusion_available,
        clinical_available,
        force_atlas_dropout=None,
        force_vascular_dropout=None,
        force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available,
            self.vascular_dropout,
            force_vascular_dropout,
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available,
            self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available,
            self.clinical_dropout,
            force_clinical_dropout,
        )
        effective_territory_valid = territory_valid.bool() & effective_atlas[:, None]
        effective_reliability = (
            territory_reliability * effective_atlas[:, None].to(
                territory_reliability.dtype
            )
        )
        effective_region_map = region_map * effective_atlas[:, None, None, None].long()
        effective_territory_features = (
            territory_features
            * effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = (
            territory_feature_valid
            * effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat(
            [
                effective_territory_features,
                effective_feature_valid,
                effective_reliability[..., None],
            ],
            dim=-1,
        )
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        
        
        
        effective_case_features[..., 0] = (
            effective_case_features[..., 0] * atlas_gate
        )
        effective_case_features[..., 1] = (
            effective_case_features[..., 1] * atlas_gate
        )
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input,
            vessel_input,
            effective_case_features,
            effective_territory_valid,
            vessel_valid.bool(),
            effective_reliability,
        )

        vascular_gate = effective_vascular.to(image.dtype)
        safe_vascular_image = (
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        v3, v4, v5, vascular_context = self.vascular_stream(
            safe_vascular_image
        )
        vascular_context = (
            vascular_context * vascular_gate[:, None]
        )
        safe_occlusion_valid = (
            occlusion_valid
            * effective_occlusion[:, None].to(occlusion_valid.dtype)
        )
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input)
        clinical_context = (
            clinical_context * effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context,
            vascular_context,
            occlusion_context,
            clinical_context,
            effective_confidence,
        )

        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](d4, territory_context, effective_region_map)
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](d3, territory_context, effective_region_map)
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](d2, territory_context, effective_region_map)
        d1 = self.decoder1(d2, e1)

        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






def center_crop_or_pad(array, output_shape, spatial_axes):
    output_array_shape = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        output_array_shape[axis] = size
    output = np.zeros(output_array_shape, dtype=array.dtype)
    source_slices = [slice(None)] * array.ndim
    destination_slices = [slice(None)] * array.ndim
    for axis, target_size in zip(spatial_axes, output_shape):
        source_size = array.shape[axis]
        copied = min(source_size, target_size)
        source_start = max((source_size - copied) // 2, 0)
        destination_start = max((target_size - copied) // 2, 0)
        source_slices[axis] = slice(source_start, source_start + copied)
        destination_slices[axis] = slice(
            destination_start, destination_start + copied
        )
    output[tuple(destination_slices)] = array[tuple(source_slices)]
    return output


def masked_scale(values, valid, mean, std):
    scaled = (values.astype(np.float32) - mean) / std
    scaled = np.clip(scaled, -SCALER_CLIP, SCALER_CLIP)
    scaled[~valid] = 0.0
    return scaled.astype(np.float32)


preflight_candidates = manifest_df[
    (manifest_df['dev_split'] == 'train')
    & manifest_df['trial8_atlas_enabled'].map(as_bool)
].sort_values('subject_id')
if len(preflight_candidates) == 0:
    raise RuntimeError('No atlas-enabled training case is available for preflight.')


preflight_row = None
for _, candidate_row in preflight_candidates.iterrows():
    candidate_npz_path = Path(str(candidate_row['vascufate_v2_npz_path']))
    with np.load(candidate_npz_path, allow_pickle=False) as candidate_data:
        candidate_lvo_present = (
            'lvo_present' in candidate_data.files
            and bool(np.asarray(candidate_data['lvo_present']).reshape(-1)[0])
        )
    if candidate_lvo_present:
        preflight_row = candidate_row
        break
if preflight_row is None:
    raise RuntimeError(
        'No atlas-enabled train-only case has a valid LVO source for the '
        'complete v2 branch smoke test.'
    )
preflight_subject = str(preflight_row['subject_id'])
npz_path = Path(str(preflight_row['vascufate_v2_npz_path']))
sidecar_path = Path(str(preflight_row['trial8_graph_sidecar_path']))
atlas_path = Path(str(preflight_row['trial8_atlas_vascular30_path']))

if file_sha256(npz_path) != str(preflight_row['trial8_npz_sha256']):
    raise RuntimeError('Preflight source NPZ hash mismatch.')
if file_sha256(sidecar_path) != str(
    preflight_row['trial8_graph_sidecar_sha256']
):
    raise RuntimeError('Preflight sidecar hash mismatch.')

with np.load(npz_path, allow_pickle=False) as data:
    required_npz_inputs = {
        'image',
        'subject_id',
        'input_channels',
        'cow_soft_union',
        'lvo_proximity',
        'lvo_present',
    }
    missing_npz_inputs = sorted(required_npz_inputs - set(data.files))
    if missing_npz_inputs:
        raise RuntimeError(
            'Preflight source NPZ lacks required input arrays: '
            + ', '.join(missing_npz_inputs)
        )
    image = np.asarray(data['image'], dtype=np.float32)
    stored_subject = scalar_string(data['subject_id'])
    input_channels = decode_strings(data['input_channels'])
    cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
    lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
    lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
if stored_subject != preflight_subject:
    raise RuntimeError('Preflight source subject mismatch.')
if input_channels != EXPECTED_INPUT_CHANNELS:
    raise RuntimeError('Preflight input-channel schema changed.')
if cow_soft_union.shape != image.shape[1:]:
    raise RuntimeError('CoW prior shape differs from the image grid.')
if lvo_proximity.shape != image.shape[1:]:
    raise RuntimeError('LVO proximity shape differs from the image grid.')

atlas_image = nib.load(str(atlas_path))
region_map = np.asanyarray(atlas_image.dataobj)
if region_map.ndim == 4 and region_map.shape[-1] == 1:
    region_map = region_map[..., 0]
region_map = np.rint(region_map).astype(np.int64)
if image.shape[1:] != region_map.shape:
    raise RuntimeError('Preflight image and atlas shapes differ.')
if region_map.min() < 0 or region_map.max() > NUM_TERRITORY_NODES:
    raise RuntimeError('Preflight atlas contains labels outside 0..30.')

with np.load(sidecar_path, allow_pickle=False) as data:
    if FORBIDDEN_TARGET_KEYS & set(data.files):
        raise RuntimeError('Preflight sidecar contains a target array.')
    node_features = np.asarray(data['node_features'], dtype=np.float32)
    node_feature_valid = np.asarray(data['node_feature_valid']) > 0
    node_valid = np.asarray(data['node_valid']) > 0
    node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
    vessel_features_source = np.asarray(data['vessel_features'], dtype=np.float32)
    vessel_feature_valid_source = np.asarray(data['vessel_feature_valid']) > 0
    vessel_valid = np.asarray(data['vessel_presence']) > 0
    case_features_source = np.asarray(data['case_features'], dtype=np.float32)

cow_prior_valid = bool(
    case_features_source[
        SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')
    ] > 0.5
)

scaled_territory = masked_scale(
    node_features,
    node_feature_valid & node_valid[:, None],
    territory_mean[None],
    territory_std[None],
)
selected_vessel = vessel_features_source[:, vessel_source_indices]
selected_vessel_valid = (
    vessel_feature_valid_source[:, vessel_source_indices] & vessel_valid[:, None]
)
scaled_vessel = masked_scale(
    selected_vessel,
    selected_vessel_valid,
    vessel_mean[None],
    vessel_std[None],
)
selected_case = case_features_source[case_source_indices].astype(np.float32)
scaled_case = selected_case.copy()
scaled_case[case_scale_mask] = np.clip(
    (scaled_case[case_scale_mask] - case_mean[case_scale_mask])
    / case_std[case_scale_mask],
    -SCALER_CLIP,
    SCALER_CLIP,
)

image_patch = center_crop_or_pad(image, PATCH_SIZE, spatial_axes=(1, 2, 3))
region_patch = center_crop_or_pad(region_map, PATCH_SIZE, spatial_axes=(0, 1, 2))
cow_patch = center_crop_or_pad(
    cow_soft_union, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
lvo_patch = center_crop_or_pad(
    lvo_proximity, PATCH_SIZE, spatial_axes=(0, 1, 2)
)
vascular_patch = np.stack([
    image_patch[input_channels.index('cta')],
    image_patch[input_channels.index('tmax')],
    image_patch[input_channels.index('perfusion_valid')],
    cow_patch,
    lvo_patch,
], axis=0).astype(np.float32)




clinical_input = np.zeros(CLINICAL_INPUT_DIM, dtype=np.float32)
clinical_input[:22] = np.linspace(-0.5, 0.5, 22, dtype=np.float32)
clinical_input[22:] = (np.arange(22) % 3 == 0).astype(np.float32)

occlusion_feature_names = [
    'cta_mean',
    'cta_p90',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
occlusion_values = [float(lvo_present), float(lvo_patch.max())]
occlusion_valid = [1.0, float(lvo_present)]
for feature_name in occlusion_feature_names:
    feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
    valid_nodes = selected_vessel_valid[:, feature_index]
    if bool(valid_nodes.any()):
        occlusion_values.append(float(scaled_vessel[valid_nodes, feature_index].mean()))
        occlusion_valid.append(1.0)
    else:
        occlusion_values.append(0.0)
        occlusion_valid.append(0.0)
occlusion_values = np.asarray(occlusion_values, dtype=np.float32)
occlusion_valid = np.asarray(occlusion_valid, dtype=np.float32)
if occlusion_values.shape != (OCCLUSION_VALUE_DIM,):
    raise RuntimeError('Occlusion-source vector dimension changed.')

confidence_features = np.asarray([
    float(vascular_patch[2].mean()),
    float(cow_prior_valid),
    float(np.clip(selected_case[1], 0.0, 1.0)),
    float(lvo_present),
], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(
        'GPU memory GB: '
        f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}'
    )

batch = {
    'image': torch.from_numpy(image_patch[None]).to(device),
    'region_map': torch.from_numpy(region_patch[None]).to(device),
    'territory_features': torch.from_numpy(scaled_territory[None]).to(device),
    'territory_feature_valid': torch.from_numpy(
        node_feature_valid.astype(np.float32)[None]
    ).to(device),
    'territory_valid': torch.from_numpy(node_valid[None]).to(device),
    'territory_reliability': torch.from_numpy(node_reliability[None]).to(device),
    'vessel_features': torch.from_numpy(scaled_vessel[None]).to(device),
    'vessel_feature_valid': torch.from_numpy(
        selected_vessel_valid.astype(np.float32)[None]
    ).to(device),
    'vessel_valid': torch.from_numpy(vessel_valid[None]).to(device),
    'case_features': torch.from_numpy(scaled_case[None]).to(device),
    'atlas_enabled': torch.ones(1, dtype=torch.bool, device=device),
    'vascular_image': torch.from_numpy(vascular_patch[None]).to(device),
    'confidence_features': torch.from_numpy(
        confidence_features[None]
    ).to(device),
    'occlusion_values': torch.from_numpy(occlusion_values[None]).to(device),
    'occlusion_valid': torch.from_numpy(occlusion_valid[None]).to(device),
    'clinical_input': torch.from_numpy(clinical_input[None]).to(device),
    'vascular_available': torch.ones(1, dtype=torch.bool, device=device),
    'occlusion_available': torch.as_tensor(
        [lvo_present], dtype=torch.bool, device=device
    ),
    'clinical_available': torch.ones(1, dtype=torch.bool, device=device),
}

model = VascuFateNetV2(
    schema=graph_schema,
    image_channels=image.shape[0],
    territory_feature_dim=len(EXPECTED_NODE_FEATURE_NAMES),
    vessel_feature_dim=len(MODEL_VESSEL_FEATURE_NAMES),
    case_feature_dim=len(MODEL_CASE_FEATURE_NAMES),
    graph_hidden=GRAPH_HIDDEN,
    graph_heads=GRAPH_HEADS,
    graph_layers=GRAPH_LAYERS,
    relation_bases=RELATION_BASES,
    graph_dropout=GRAPH_DROPOUT,
    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
    film_bound=FILM_BOUND,
    vascular_channels=VASCULAR_CHANNELS,
    clinical_input_dim=CLINICAL_INPUT_DIM,
    clinical_embed_dim=CLINICAL_EMBED_DIM,
    occlusion_value_dim=OCCLUSION_VALUE_DIM,
    occlusion_embed_dim=OCCLUSION_EMBED_DIM,
    confidence_dim=CONFIDENCE_DIM,
    fusion_bound=FUSION_BOUND,
).to(device)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


def parameter_count(module, trainable_only=False):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if not trainable_only or parameter.requires_grad
    )


total_parameters = parameter_count(model)
trainable_parameters = parameter_count(model, trainable_only=True)
graph_parameters = parameter_count(model.graph_encoder)
film_parameters = parameter_count(model.global_film) + parameter_count(
    model.regional_films
)
head_parameters = (
    parameter_count(model.segmentation_head)
    + parameter_count(model.auxiliary_half_head)
    + parameter_count(model.auxiliary_quarter_head)
)
vascular_stream_parameters = parameter_count(model.vascular_stream)
spatial_fusion_parameters = parameter_count(
    model.vascular_fusion_quarter
) + parameter_count(model.vascular_fusion_eighth) + parameter_count(
    model.vascular_fusion_bottleneck
)
clinical_parameters = parameter_count(model.clinical_encoder)
occlusion_parameters = parameter_count(model.occlusion_encoder)
context_fusion_parameters = parameter_count(model.context_fusion)
new_fusion_parameters = (
    vascular_stream_parameters
    + spatial_fusion_parameters
    + clinical_parameters
    + occlusion_parameters
    + context_fusion_parameters
)
backbone_parameters = total_parameters - (
    graph_parameters
    + film_parameters
    + head_parameters
    + new_fusion_parameters
)
preserved_v1_parameters = (
    backbone_parameters + graph_parameters + film_parameters + head_parameters
)

print('\nTrial8 parameter count:')
display(pd.DataFrame([
    {'component': 'image backbone', 'parameters': backbone_parameters},
    {'component': 'heterogeneous RA-GATv2', 'parameters': graph_parameters},
    {'component': 'bounded zero-init FiLM', 'parameters': film_parameters},
    {'component': 'segmentation/deep-supervision heads', 'parameters': head_parameters},
    {'component': 'asymmetric vascular stream', 'parameters': vascular_stream_parameters},
    {'component': 'confidence-gated spatial fusion', 'parameters': spatial_fusion_parameters},
    {'component': 'clinical token encoder', 'parameters': clinical_parameters},
    {'component': 'occlusion source encoder', 'parameters': occlusion_parameters},
    {'component': 'gated context fusion', 'parameters': context_fusion_parameters},
    {'component': 'preserved v1 path', 'parameters': preserved_v1_parameters},
    {'component': 'total', 'parameters': total_parameters},
    {'component': 'trainable', 'parameters': trainable_parameters},
]))






autocast_enabled = device.type == 'cuda'
model.eval()
with torch.no_grad():
    normal_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    atlas_dropped_output = model(
        **batch,
        force_atlas_dropout=True,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    new_sources_dropped_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=True,
        force_clinical_dropout=True,
    )

expected_shapes = {
    'logits': (1, 1, *PATCH_SIZE),
    'auxiliary_half_logits': (1, 1, PATCH_SIZE[0] // 2, PATCH_SIZE[1] // 2, PATCH_SIZE[2] // 2),
    'auxiliary_quarter_logits': (1, 1, PATCH_SIZE[0] // 4, PATCH_SIZE[1] // 4, PATCH_SIZE[2] // 4),
    'territory_node_logits': (1, NUM_TERRITORY_NODES),
    'global_graph_context': (1, GRAPH_HIDDEN),
    'fused_global_context': (1, GRAPH_HIDDEN),
    'new_source_context': (1, GRAPH_HIDDEN),
}

forward_checks = {
    f'{name}_shape_matches': tuple(normal_output[name].shape) == shape
    for name, shape in expected_shapes.items()
}
forward_checks.update({
    'normal_outputs_finite': all(
        torch.isfinite(normal_output[name]).all().item()
        for name in expected_shapes
    ),
    'forced_dropout_outputs_finite': all(
        torch.isfinite(new_sources_dropped_output[name]).all().item()
        for name in expected_shapes
    ),
    'normal_atlas_enabled': bool(
        normal_output['effective_atlas_enabled'].item()
    ),
    'forced_atlas_dropout_applied': bool(
        atlas_dropped_output['atlas_dropped'].item()
        and not atlas_dropped_output['effective_atlas_enabled'].item()
    ),
    'initial_segmentation_is_graph_neutral': torch.allclose(
        normal_output['logits'],
        atlas_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'initial_segmentation_is_new_sources_neutral': torch.allclose(
        normal_output['logits'],
        new_sources_dropped_output['logits'],
        atol=0.0,
        rtol=0.0,
    ),
    'zero_init_film_parameters_are_zero': all(
        (
            torch.count_nonzero(module.project[-1].weight).item() == 0
            and torch.count_nonzero(module.project[-1].bias).item() == 0
        )
        if isinstance(module, GlobalFiLM3D)
        else (
            torch.count_nonzero(module.project.weight).item() == 0
            and torch.count_nonzero(module.project.bias).item() == 0
        )
        for module in [model.global_film, *model.regional_films]
    ),
    'zero_init_new_fusion_gates_are_zero': all(
        torch.count_nonzero(module.network[-1].weight).item() == 0
        and torch.count_nonzero(module.network[-1].bias).item() == 0
        for module in [
            model.vascular_fusion_quarter.gate,
            model.vascular_fusion_eighth.gate,
            model.vascular_fusion_bottleneck.gate,
            model.context_fusion.gate,
        ]
    ),
    'normal_vascular_source_enabled': bool(
        normal_output['effective_vascular_enabled'].item()
    ),
    'normal_clinical_source_enabled': bool(
        normal_output['effective_clinical_enabled'].item()
    ),
    'forced_new_source_dropout_applied': bool(
        new_sources_dropped_output['vascular_dropped'].item()
        and new_sources_dropped_output['occlusion_dropped'].item()
        and new_sources_dropped_output['clinical_dropped'].item()
    ),
})




model.activate_new_fusion_gates_for_gradient_test()
with torch.no_grad():
    nn.init.normal_(model.global_film.project[-1].weight, mean=0.0, std=1e-3)
    nn.init.zeros_(model.global_film.project[-1].bias)

model.eval()
with torch.no_grad():
    activated_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    vascular_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=True,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    occlusion_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=True,
        force_clinical_dropout=False,
    )
    clinical_ablation_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=True,
    )

ablation_differences = {
    'vascular_max_abs_logit_change': float(
        (activated_output['logits'] - vascular_ablation_output['logits'])
        .abs().max().item()
    ),
    'occlusion_max_abs_logit_change': float(
        (activated_output['logits'] - occlusion_ablation_output['logits'])
        .abs().max().item()
    ),
    'clinical_max_abs_logit_change': float(
        (activated_output['logits'] - clinical_ablation_output['logits'])
        .abs().max().item()
    ),
}
ablation_checks = {
    'activated_outputs_finite': all(
        torch.isfinite(activated_output[name]).all().item()
        for name in expected_shapes
    ),
    'vascular_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['vascular_max_abs_logit_change'] > 1e-8
    ),
    'occlusion_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['occlusion_max_abs_logit_change'] > 1e-8
    ),
    'clinical_ablation_changes_logits_after_controlled_activation': (
        ablation_differences['clinical_max_abs_logit_change'] > 1e-8
    ),
    'all_activated_gates_remain_bounded': all(
        float(activated_output[name].abs().max().item()) <= FUSION_BOUND + 1e-6
        for name in [
            'context_fusion_gate',
            'vascular_quarter_gate',
            'vascular_eighth_gate',
            'vascular_bottleneck_gate',
        ]
    ),
}

model.train()
model.zero_grad(set_to_none=True)
with torch.autocast(
    device_type=device.type,
    dtype=torch.float16 if autocast_enabled else torch.bfloat16,
    enabled=False,
):
    gradient_output = model(
        **batch,
        force_atlas_dropout=False,
        force_vascular_dropout=False,
        force_occlusion_dropout=False,
        force_clinical_dropout=False,
    )
    smoke_loss = (
        gradient_output['logits'].float().square().mean()
        + 0.15
        * gradient_output['auxiliary_half_logits'].float().square().mean()
        + 0.10
        * gradient_output['auxiliary_quarter_logits'].float().square().mean()
        + 0.05
        * gradient_output['territory_node_logits'].float().square().mean()
    )
smoke_loss.backward()

gradient_norms = {}
for group_name, module in [
    ('image_backbone', model.stem),
    ('graph_encoder', model.graph_encoder),
    ('global_film', model.global_film),
    ('regional_film', model.regional_films),
    ('vascular_stream', model.vascular_stream),
    ('spatial_fusion', nn.ModuleList([
        model.vascular_fusion_quarter,
        model.vascular_fusion_eighth,
        model.vascular_fusion_bottleneck,
    ])),
    ('clinical_encoder', model.clinical_encoder),
    ('occlusion_encoder', model.occlusion_encoder),
    ('context_fusion', model.context_fusion),
    ('segmentation_head', model.segmentation_head),
]:
    squared = 0.0
    gradient_parameters = 0
    for parameter in module.parameters():
        if parameter.grad is not None:
            squared += float(parameter.grad.detach().float().square().sum().item())
            gradient_parameters += parameter.numel()
    gradient_norms[group_name] = math.sqrt(squared)
    gradient_norms[f'{group_name}_gradient_parameters'] = gradient_parameters

gradient_checks = {
    'smoke_loss_finite': bool(torch.isfinite(smoke_loss).item()),
    'image_backbone_has_gradients': gradient_norms['image_backbone'] > 0,
    'graph_encoder_has_gradients': gradient_norms['graph_encoder'] > 0,
    'global_film_has_gradients': gradient_norms['global_film'] > 0,
    'regional_film_has_gradients': gradient_norms['regional_film'] > 0,
    'vascular_stream_has_gradients_after_gate_activation': (
        gradient_norms['vascular_stream'] > 0
    ),
    'spatial_fusion_has_gradients_after_gate_activation': (
        gradient_norms['spatial_fusion'] > 0
    ),
    'clinical_encoder_has_gradients_after_gate_activation': (
        gradient_norms['clinical_encoder'] > 0
    ),
    'occlusion_encoder_has_gradients_after_gate_activation': (
        gradient_norms['occlusion_encoder'] > 0
    ),
    'context_fusion_has_gradients_after_gate_activation': (
        gradient_norms['context_fusion'] > 0
    ),
    'segmentation_head_has_gradients': gradient_norms['segmentation_head'] > 0,
}

model.zero_grad(set_to_none=True)
if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
else:
    peak_memory_gb = 0.0

architecture_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'image_channels': int(image.shape[0]),
    'patch_size': PATCH_SIZE,
    'backbone_channels': [32, 64, 128, 224, 320],
    'baseline_architecture_preserved': True,
    'baseline_parameter_count': EXPECTED_BASELINE_PARAMETERS,
    'graph_variant': 'two_layer_relation_aware_gated_GATv2',
    'graph_hidden': GRAPH_HIDDEN,
    'graph_heads': GRAPH_HEADS,
    'graph_layers': GRAPH_LAYERS,
    'relation_bases': RELATION_BASES,
    'graph_dropout': GRAPH_DROPOUT,
    'edge_weight_use': 'log_attention_prior',
    'territory_nodes': NUM_TERRITORY_NODES,
    'vessel_nodes': NUM_VESSEL_NODES,
    'atlas_dropout_probability': ATLAS_DROPOUT_PROBABILITY,
    'conditioning': (
        'preserved_bounded_zero_initialized_multiscale_FiLM_plus_'
        'confidence_gated_zero_initialized_vascular_context_fusion'
    ),
    'film_bound': FILM_BOUND,
    'fusion_bound': FUSION_BOUND,
    'asymmetric_vascular_stream': {
        'input_channels': [
            'cta', 'tmax', 'perfusion_valid',
            'cow_soft_union', 'lvo_proximity',
        ],
        'channels': VASCULAR_CHANNELS,
        'fusion_scales': ['quarter', 'eighth', 'sixteenth'],
        'initial_gate': 'exact_zero',
    },
    'clinical_token': {
        'input_dimension': CLINICAL_INPUT_DIM,
        'embedding_dimension': CLINICAL_EMBED_DIM,
        'values': 22,
        'paired_missingness_indicators': 22,
        'fold_specific_scaler_required_before_training': True,
    },
    'occlusion_source': {
        'value_dimension': OCCLUSION_VALUE_DIM,
        'embedding_dimension': OCCLUSION_EMBED_DIM,
        'validity_masked': True,
        'uses_tmax_not_ttp': True,
    },
    'confidence_features': [
        'perfusion_valid_fraction',
        'cow_prior_valid',
        'registration_confidence',
        'lvo_present',
    ],
    'staged_neutrality': {
        'new_fusion_gates_exactly_zero_at_initialization': True,
        'controlled_gate_activation_used_only_for_preflight': True,
        'saved_or_trained_weights_created': False,
    },
    'graph_to_logit_residual': False,
    'expert_router': False,
    'deep_supervision': True,
    'territory_auxiliary_head': True,
    'scaler_policy': 'fit_on_train_only_then_freeze',
    'expected_total_parameters': EXPECTED_TOTAL_PARAMETERS,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'component_parameters': {
        'image_backbone': backbone_parameters,
        'heterogeneous_graph': graph_parameters,
        'film': film_parameters,
        'heads': head_parameters,
        'asymmetric_vascular_stream': vascular_stream_parameters,
        'spatial_fusion': spatial_fusion_parameters,
        'clinical_encoder': clinical_parameters,
        'occlusion_encoder': occlusion_parameters,
        'context_fusion': context_fusion_parameters,
        'new_fusion_total': new_fusion_parameters,
        'preserved_v1_total': preserved_v1_parameters,
    },
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_sha256': scaler_sha256,
    'feature_availability_gate_path': str(AVAILABILITY_GATE_PATH),
    'architecture_input_contract_path': str(INPUT_CONTRACT_PATH),
    'ablation_differences': ablation_differences,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'model_training_authorized': False,
}
atomic_write_json(architecture_config, ARCHITECTURE_JSON_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)

preflight_checks = {
    **manifest_checks,
    **scaler_checks,
    **forward_checks,
    **ablation_checks,
    **gradient_checks,
    'feature_availability_audit_passed': (
        availability_gate.get('feature_availability_audit_passed') is True
    ),
    'input_contract_authorizes_architecture_implementation': (
        input_contract.get('architecture_implementation_authorized') is True
    ),
    'preserved_v1_parameter_count_matches': (
        preserved_v1_parameters == EXPECTED_BASELINE_PARAMETERS
    ),
    'parameter_count_in_frozen_efficiency_budget': (
        PARAMETER_BUDGET_MINIMUM
        <= total_parameters
        <= PARAMETER_BUDGET_MAXIMUM
    ),
    'parameter_count_matches_frozen_architecture': total_parameters
    == EXPECTED_TOTAL_PARAMETERS,
    'new_fusion_parameter_count_is_bounded': (
        0 < new_fusion_parameters < 600_000
    ),
    'all_parameters_trainable_at_initialization': total_parameters
    == trainable_parameters,
    'peak_memory_below_l4_capacity': peak_memory_gb < 20.0
    if device.type == 'cuda'
    else True,
    'target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'direct_graph_to_logit_residual_disabled': True,
    'expert_routing_disabled': True,
    'training_remained_locked': True,
}
preflight_passed = bool(all(preflight_checks.values()))

audit_rows = [
    {'category': 'manifest', 'check': name, 'passed': passed}
    for name, passed in manifest_checks.items()
] + [
    {'category': 'scaler', 'check': name, 'passed': passed}
    for name, passed in scaler_checks.items()
] + [
    {'category': 'forward', 'check': name, 'passed': passed}
    for name, passed in forward_checks.items()
] + [
    {'category': 'ablation', 'check': name, 'passed': passed}
    for name, passed in ablation_checks.items()
] + [
    {'category': 'gradient', 'check': name, 'passed': passed}
    for name, passed in gradient_checks.items()
 ] + [
    {'category': 'final_gate', 'check': name, 'passed': passed}
    for name, passed in preflight_checks.items()
    if name not in {
        *manifest_checks.keys(),
        *scaler_checks.keys(),
        *forward_checks.keys(),
        *ablation_checks.keys(),
        *gradient_checks.keys(),
    }
]
atomic_write_csv(pd.DataFrame(audit_rows), PREFLIGHT_AUDIT_PATH)

gate_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'preflight_passed': preflight_passed,
    'checks': preflight_checks,
    'preflight_subject': preflight_subject,
    'verified_manifest_path': str(VERIFIED_MANIFEST_PATH),
    'verified_manifest_sha256': verified_manifest_sha256,
    'graph_schema_path': str(SAFE_GRAPH_SCHEMA_PATH),
    'graph_schema_sha256': safe_schema_sha256,
    'scaler_path': str(SCALER_NPZ_PATH),
    'scaler_sha256': scaler_sha256,
    'architecture_config_path': str(ARCHITECTURE_JSON_PATH),
    'architecture_config_sha256': architecture_sha256,
    'total_parameters': total_parameters,
    'trainable_parameters': trainable_parameters,
    'smoke_loss': float(smoke_loss.detach().cpu().item()),
    'gradient_norms': gradient_norms,
    'ablation_differences': ablation_differences,
    'peak_gpu_memory_gb': peak_memory_gb,
    'target_arrays_read': False,
    'outcome_workbooks_read': False,
    'controlled_gate_activation_was_diagnostic_only': True,
    'saved_or_trained_weights_created': False,
    'v2_loader_and_clinical_scaler_implementation_authorized': preflight_passed,
    'model_training_authorized': False,
}
atomic_write_json(gate_record, PREFLIGHT_GATE_PATH)

print('\nModel preflight checks:')
display(pd.DataFrame([
    {'check': name, 'passed': passed} for name, passed in preflight_checks.items()
]))
print('\nOutput shapes:')
display(pd.DataFrame([
    {
        'output': name,
        'observed_shape': str(tuple(normal_output[name].shape)),
        'expected_shape': str(expected_shapes[name]),
    }
    for name in expected_shapes
]))
print('\nGradient norms:')
display(pd.DataFrame([
    {'component': name, 'gradient_norm': value}
    for name, value in gradient_norms.items()
    if not name.endswith('_gradient_parameters')
]))

if not preflight_passed:
    failed = [name for name, passed in preflight_checks.items() if not passed]
    raise RuntimeError(
        'TRIAL8 V2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not start training. '
        'Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 VASCUFATE-NET V2 MODEL CONSTRUCTION PREFLIGHT PASSED')
print('=' * 80)
print(f'Preflight subject:                  {preflight_subject}')
print(f'Input patch:                        {(1, image.shape[0], *PATCH_SIZE)}')
print(f'Total parameters:                   {total_parameters:,}')
print(f'Trainable parameters:               {trainable_parameters:,}')
print(f'Image-backbone parameters:          {backbone_parameters:,}')
print(f'Heterogeneous graph parameters:     {graph_parameters:,}')
print(f'FiLM parameters:                    {film_parameters:,}')
print(f'Head parameters:                    {head_parameters:,}')
print(f'New fusion parameters:              {new_fusion_parameters:,}')
print(f'  Asymmetric vascular stream:       {vascular_stream_parameters:,}')
print(f'  Spatial confidence fusion:        {spatial_fusion_parameters:,}')
print(f'  Clinical encoder:                 {clinical_parameters:,}')
print(f'  Occlusion encoder:                {occlusion_parameters:,}')
print(f'  Context fusion:                   {context_fusion_parameters:,}')
print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
print('Expert routing:                     False')
print('Graph-to-logit residual:            False')
print('Atlas dropout verified:             True')
print('New fusion initially neutral:       True')
print('Controlled ablations responsive:    True')
print('Graph scaler fitted on:             train only (102 subjects)')
print('Clinical scaler fitted:             False (next preflight)')
print('Target arrays read:                 False')
print('Outcome workbooks read:              False')
print('Model training authorized:          False')
print(f'V2 loader implementation allowed:   {preflight_passed}')
print(f'Scaler:                             {SCALER_NPZ_PATH}')
print(f'Architecture:                       {ARCHITECTURE_JSON_PATH}')
print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
print(
    '\nThe standalone block will now verify the completed multimodal-loader '
    'gate and continue into the isolated fold-0 diagnostic.'
)






PILOT_CODE_REVISION = 'r1_branch_component_threshold_overfit_audit'
PILOT_VERSION = 'v2_2_fold0_failure_diagnostic_r1'
LOADER_PREFLIGHT_VERSION = 'v1_fold_clinical_scalers_multimodal_loader'
FIVEFOLD_PREFLIGHT_VERSION = 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_V2_PARAMETERS = 15_428_468
HELDOUT_FOLD = 0
PILOT_SEED = 20260928
PILOT_PATCH_SIZE = (128, 128, 64)
NUMBER_OF_FOLDS = 5
MINIMUM_POSITIVE_PATCHES_PER_BIN = 20
MAXIMUM_NEGATIVE_CENTER_ATTEMPTS = 512
MAXIMUM_CENTER_RETRIES = 16
POSITIVE_BINS = ['tiny', 'small', 'medium', 'large', 'very_large']
PILOT_CLINICAL_FEATURE_NAMES = [
    'sex_female',
    'age_years',
    'atrial_fibrillation',
    'hypertension',
    'diabetes',
    'hyperlipidemia',
    'anticoagulation',
    'lipid_lowering_drugs',
    'platelet_aggregation_inhibitors',
    'glucose_mg_dl',
    'leucocytes_g_l',
    'crp_mg_dl',
    'inr',
    'wake_up_stroke',
    'in_house_stroke',
    'external_referral',
    'onset_to_door_hours',
    'alert_to_door_hours',
    'nihss_at_admission',
    'mrs_at_admission',
    'mrs_premorbid',
    'door_to_imaging_hours',
]

MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
LOADER_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / MODEL_VERSION
    / LOADER_PREFLIGHT_VERSION
)
LOADER_GATE_PATH = LOADER_PREFLIGHT_ROOT / 'v2_multimodal_loader_gate.json'
FIVEFOLD_PREFLIGHT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / BASELINE_MODEL_VERSION
    / FIVEFOLD_PREFLIGHT_VERSION
)
FIVEFOLD_GATE_PATH = (
    FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_loader_loss_sampler_gate.json'
)
FIVEFOLD_POLICY_PATH = FIVEFOLD_PREFLIGHT_ROOT / 'fivefold_training_policy.json'

PILOT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / MODEL_VERSION
    / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_START_GATE_PATH = PILOT_ROOT / 'fold0_pilot_start_gate.json'
PILOT_CONFIG_PATH = PILOT_ROOT / 'fold0_pilot_config.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'fold0_training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'fold0_pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'fold0_pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'fold0_epoch1_schedule_audit.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_json_contract_hash(record, excluded_top_level_keys=()):
    excluded = set(excluded_top_level_keys)
    stable_record = {
        key: value for key, value in record.items() if key not in excluded
    }
    payload = json.dumps(
        to_builtin(stable_record),
        sort_keys=True,
        separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


pilot_required_paths = [
    MULTIMODAL_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    LOADER_GATE_PATH,
    FIVEFOLD_GATE_PATH,
    FIVEFOLD_POLICY_PATH,
    SAFE_GRAPH_SCHEMA_PATH,
    ARCHITECTURE_JSON_PATH,
    PREFLIGHT_GATE_PATH,
]
pilot_missing_paths = [str(path) for path in pilot_required_paths if not path.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Fold-0 pilot prerequisites are missing:\n' + '\n'.join(pilot_missing_paths)
    )

multimodal_manifest_sha256 = file_sha256(MULTIMODAL_MANIFEST_PATH)
fold_manifest_sha256 = file_sha256(FOLD_MANIFEST_PATH)
loader_gate_sha256 = file_sha256(LOADER_GATE_PATH)
fivefold_gate_sha256 = file_sha256(FIVEFOLD_GATE_PATH)
architecture_sha256 = file_sha256(ARCHITECTURE_JSON_PATH)
v2_model_gate_sha256 = file_sha256(PREFLIGHT_GATE_PATH)
architecture_contract_sha256 = pilot_json_contract_hash(
    architecture_config,
    excluded_top_level_keys={'created_at_utc', 'ablation_differences'},
)
if multimodal_manifest_sha256 != EXPECTED_MULTIMODAL_MANIFEST_SHA256:
    raise RuntimeError('Multimodal candidate manifest hash changed.')
if fold_manifest_sha256 != EXPECTED_FOLD_MANIFEST_SHA256:
    raise RuntimeError('Five-fold manifest hash changed.')

loader_gate = json.loads(LOADER_GATE_PATH.read_text(encoding='utf-8'))
fivefold_gate = json.loads(FIVEFOLD_GATE_PATH.read_text(encoding='utf-8'))
fivefold_policy = json.loads(FIVEFOLD_POLICY_PATH.read_text(encoding='utf-8'))
v2_model_gate = json.loads(PREFLIGHT_GATE_PATH.read_text(encoding='utf-8'))

pilot_upstream_checks = {
    'v2_model_preflight_passed': v2_model_gate.get('preflight_passed') is True,
    'v2_model_parameter_count_matches': int(
        v2_model_gate.get('total_parameters', -1)
    ) == EXPECTED_V2_PARAMETERS,
    'v2_model_architecture_hash_matches': (
        v2_model_gate.get('architecture_config_sha256') == architecture_sha256
    ),
    'multimodal_loader_preflight_passed': (
        loader_gate.get('preflight_passed') is True
    ),
    'multimodal_manifest_hash_matches_loader_gate': (
        loader_gate.get('candidate_manifest_sha256')
        == multimodal_manifest_sha256
    ),
    'fold_manifest_hash_matches_loader_gate': (
        loader_gate.get('fold_manifest_sha256') == fold_manifest_sha256
    ),
    'graph_schema_hash_matches_loader_gate': (
        loader_gate.get('graph_schema_sha256') == safe_schema_sha256
    ),
    'pilot_implementation_authorized': (
        loader_gate.get('pilot_training_implementation_authorized') is True
    ),
    'upstream_training_remained_locked': (
        loader_gate.get('fold0_pilot_training_authorized') is False
        and loader_gate.get('full_fivefold_training_authorized') is False
    ),
    'loader_target_free': loader_gate.get('target_arrays_read') is False,
    'loader_outcome_free': loader_gate.get('outcome_workbooks_read') is False,
    'fivefold_loader_loss_sampler_passed': (
        fivefold_gate.get('preflight_passed') is True
        and fivefold_gate.get(
            'baseline_loader_loss_sampler_contract_verified'
        ) is True
    ),
    'fivefold_policy_training_locked': (
        fivefold_gate.get('fold0_pilot_training_authorized') is False
        and fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed_upstream': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed_upstream': fivefold_gate.get('test_targets_read') is False,
    'architecture_still_has_no_expert_router': (
        architecture_config.get('expert_router') is False
    ),
    'architecture_still_has_no_graph_logit_residual': (
        architecture_config.get('graph_to_logit_residual') is False
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [name for name, passed in pilot_upstream_checks.items() if not passed]
    raise RuntimeError('Fold-0 pilot upstream checks failed: ' + ', '.join(failed))

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].astype(str).tolist() != pilot_fold_df[
    'subject_id'
].astype(str).tolist():
    raise RuntimeError('Multimodal and fold manifests have different subjects.')
if 'development_fold' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_fold'] = pilot_fold_df[
        'development_fold'
    ].astype(int)
if 'development_role' not in pilot_manifest_df.columns:
    pilot_manifest_df['development_role'] = pilot_fold_df[
        'development_role'
    ].astype(str)

pilot_required_columns = {
    'subject_id', 'dev_split', 'development_fold', 'vascufate_v2_npz_path',
    'trial8_npz_sha256', 'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
    'trial8_clinical_sidecar_sha256',
    'trial8_v2_multimodal_loader_preflight_passed',
    'trial8_model_training_authorized', 'trial8_supervision_mode',
    'trial8_effective_target_volume_ml',
    'trial8_effective_lesion_volume_bin', 'trial8_sampling_positive',
    'trial8_hard_metric_eligible',
}
pilot_missing_columns = sorted(
    pilot_required_columns - set(pilot_manifest_df.columns)
)
if pilot_missing_columns:
    raise RuntimeError(
        'Multimodal manifest is missing columns: ' + ', '.join(pilot_missing_columns)
    )

development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'].astype(int) != HELDOUT_FOLD
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'].astype(int) == HELDOUT_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()

pilot_manifest_checks = {
    'manifest_has_148_subjects': len(pilot_manifest_df) == 148,
    'manifest_subjects_unique': pilot_manifest_df['subject_id'].nunique() == 148,
    'split_counts_match': pilot_manifest_df['dev_split'].value_counts().to_dict()
    == EXPECTED_SPLITS,
    'loader_preflight_passed_for_all': bool(
        pilot_manifest_df[
            'trial8_v2_multimodal_loader_preflight_passed'
        ].map(as_bool).all()
    ),
    'manifest_training_remained_locked': not bool(
        pilot_manifest_df['trial8_model_training_authorized'].map(as_bool).any()
    ),
    'fold0_fit_has_82_subjects': len(fold0_fit_df) == 82,
    'fold0_heldout_has_20_subjects': len(fold0_heldout_df) == 20,
    'locked_validation_has_23_subjects': len(locked_validation_df) == 23,
    'sealed_test_has_23_subjects': len(sealed_test_df) == 23,
    'fold0_fit_and_heldout_are_disjoint': not bool(
        set(fold0_fit_df['subject_id']) & set(fold0_heldout_df['subject_id'])
    ),
}
if not all(pilot_manifest_checks.values()):
    failed = [name for name, passed in pilot_manifest_checks.items() if not passed]
    raise RuntimeError('Fold-0 manifest checks failed: ' + ', '.join(failed))

fold0_graph_scaler_path = Path(loader_gate['graph_scaler_paths'][str(HELDOUT_FOLD)])
fold0_graph_scaler_sha256 = loader_gate['graph_scaler_sha256'][
    str(HELDOUT_FOLD)
]
fold0_clinical_scaler_path = Path(
    loader_gate['clinical_scaler_paths'][str(HELDOUT_FOLD)]
)
fold0_clinical_scaler_sha256 = loader_gate['clinical_scaler_sha256'][
    str(HELDOUT_FOLD)
]
for scaler_path, expected_hash, label in [
    (fold0_graph_scaler_path, fold0_graph_scaler_sha256, 'graph'),
    (fold0_clinical_scaler_path, fold0_clinical_scaler_sha256, 'clinical'),
]:
    if not scaler_path.exists():
        raise FileNotFoundError(f'Fold-0 {label} scaler is missing: {scaler_path}')
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-0 {label} scaler hash changed.')

with np.load(fold0_graph_scaler_path, allow_pickle=False) as data:
    fold0_graph_scaler = {name: np.asarray(data[name]) for name in data.files}
with np.load(fold0_clinical_scaler_path, allow_pickle=False) as data:
    fold0_clinical_scaler = {name: np.asarray(data[name]) for name in data.files}

if decode_strings(fold0_graph_scaler['territory_feature_names']) != (
    EXPECTED_NODE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 territory feature order changed.')
if decode_strings(fold0_graph_scaler['vessel_feature_names']) != (
    MODEL_VESSEL_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 vessel feature order changed.')
if decode_strings(fold0_graph_scaler['case_feature_names']) != (
    MODEL_CASE_FEATURE_NAMES
):
    raise RuntimeError('Fold-0 case feature order changed.')
if decode_strings(
    fold0_clinical_scaler['feature_names']
) != PILOT_CLINICAL_FEATURE_NAMES:
    raise RuntimeError('Fold-0 clinical feature order changed.')

fold0_fit_subject_hash = pilot_subject_hash(fold0_fit_df['subject_id'])
fold0_heldout_subject_hash = pilot_subject_hash(fold0_heldout_df['subject_id'])
pilot_scaler_contract_checks = {
    'graph_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_graph_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'graph_scaler_fit_count_is_82': int(
        np.asarray(fold0_graph_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'graph_scaler_fit_subject_hash_matches': scalar_string(
        fold0_graph_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'graph_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_graph_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'clinical_scaler_heldout_fold_is_zero': int(
        np.asarray(fold0_clinical_scaler['heldout_fold']).reshape(-1)[0]
    ) == HELDOUT_FOLD,
    'clinical_scaler_fit_count_is_82': int(
        np.asarray(fold0_clinical_scaler['fit_subject_count']).reshape(-1)[0]
    ) == 82,
    'clinical_scaler_heldout_count_is_20': int(
        np.asarray(fold0_clinical_scaler['heldout_subject_count']).reshape(-1)[0]
    ) == 20,
    'clinical_scaler_fit_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['fit_subject_hash']
    ) == fold0_fit_subject_hash,
    'clinical_scaler_heldout_subject_hash_matches': scalar_string(
        fold0_clinical_scaler['heldout_subject_hash']
    ) == fold0_heldout_subject_hash,
    'graph_scaler_is_target_free': not bool(
        np.asarray(fold0_graph_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_target_free': not bool(
        np.asarray(fold0_clinical_scaler['target_arrays_read']).reshape(-1)[0]
    ),
    'clinical_scaler_is_outcome_free': not bool(
        np.asarray(
            fold0_clinical_scaler['outcome_workbooks_read']
        ).reshape(-1)[0]
    ),
}
if not all(pilot_scaler_contract_checks.values()):
    failed = [
        name for name, passed in pilot_scaler_contract_checks.items()
        if not passed
    ]
    raise RuntimeError('Fold-0 scaler contract failed: ' + ', '.join(failed))

print('\n' + '=' * 80)
print('TRIAL8 V2.2 FOLD-0 DIAGNOSTIC CONTRACT PASSED')
print('=' * 80)
print(f'Diagnostic code revision:            {PILOT_CODE_REVISION}')
print(f'Fold-0 fit / held-out:               {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
print(f'Locked validation / sealed test:     {len(locked_validation_df)} / {len(sealed_test_df)}')
print('Target access authorized:            development train split only')
print('Locked validation targets read:      False')
print('Test targets read:                   False')


for disposable_name in [
    'model', 'batch', 'normal_output', 'forced_atlas_output',
    'forced_vascular_output', 'forced_occlusion_output',
    'forced_clinical_output', 'gradient_output', 'smoke_loss',
]:
    globals().pop(disposable_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(value) for value in mask.shape),
        'packed': np.packbits(mask.reshape(-1)),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    return np.unpackbits(record['packed'], count=count).reshape(
        record['shape']
    ).astype(bool)


fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')
all_development_row_by_subject = {
    str(row['subject_id']): row for row in development_df.to_dict('records')
}
pilot_mask_records = {}
pilot_shape_by_subject = {}
pilot_target_rows = []
for row in development_df.sort_values('subject_id').to_dict('records'):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen NPZ hash changed.')
    with np.load(npz_path, allow_pickle=False) as data:
        required_target_arrays = {
            'target_binary', 'target_soft', 'binary_loss_weight',
            'crop_mask', 'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required_target_arrays - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing: {missing}')
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        hard_weight = pilot_scalar(data, 'binary_loss_weight')
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
        source_volume_ml = pilot_scalar(data, 'source_target_volume_ml')
    if target_binary.shape != target_soft.shape or target_binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes differ.')
    if not np.isfinite(target_soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target arrays are non-finite.')
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        sampling_support = target_binary
        expected_hard_weight = 1.0
    elif supervision_mode == 'soft_only_subvoxel':
        sampling_support = target_soft > 0
        expected_hard_weight = 0.0
    elif supervision_mode == 'true_empty':
        sampling_support = np.zeros_like(target_binary)
        expected_hard_weight = 1.0
    else:
        raise RuntimeError(f'{subject_id}: invalid supervision mode.')
    exclusion_support = target_binary | (target_soft > 0)
    pilot_shape_by_subject[subject_id] = tuple(
        int(value) for value in target_binary.shape
    )
    if not math.isclose(hard_weight, expected_hard_weight):
        raise RuntimeError(f'{subject_id}: hard-loss policy changed.')
    if subject_id in fit_row_by_subject:
        pilot_mask_records[subject_id] = {
            'sampling_support': pilot_pack_mask(sampling_support),
            'exclusion_support': pilot_pack_mask(exclusion_support),
            'crop_mask': pilot_pack_mask(crop_mask),
        }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_fold': int(row['development_fold']),
        'pilot_role': 'fit' if subject_id in fit_row_by_subject else 'heldout',
        'supervision_mode': supervision_mode,
        'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
        'sampling_support_voxels': int(sampling_support.sum()),
        'binary_voxels': int(target_binary.sum()),
        'soft_nonzero_voxels': int((target_soft > 0).sum()),
        'source_target_volume_ml': source_volume_ml,
        'hard_weight': hard_weight,
        'voxel_volume_ml': float(np.prod(spacing) / 1000.0),
        'target_arrays_read_split': 'train',
    })
pilot_target_df = pd.DataFrame(pilot_target_rows)
atomic_write_csv(pilot_target_df, PILOT_TARGET_AUDIT_PATH)
pilot_target_by_subject = pilot_target_df.set_index('subject_id').to_dict('index')

pilot_target_checks = {
    'exactly_102_development_targets_read': len(pilot_target_df) == 102,
    'target_access_roles_are_fit_or_heldout': set(
        pilot_target_df['pilot_role']
    ) == {'fit', 'heldout'},
    'target_access_is_train_split_only': set(
        pilot_target_df['target_arrays_read_split']
    ) == {'train'},
    'fit_sampling_masks_cover_82_subjects': len(pilot_mask_records) == 82,
    'soft_only_hard_loss_disabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'] == 'soft_only_subvoxel',
            'hard_weight',
        ].eq(0.0).all()
    ),
    'hard_and_empty_hard_loss_enabled': bool(
        pilot_target_df.loc[
            pilot_target_df['supervision_mode'].isin(
                ['hard_and_soft', 'true_empty']
            ),
            'hard_weight',
        ].eq(1.0).all()
    ),
    'locked_validation_targets_not_read': True,
    'test_targets_not_read': True,
}
if not all(pilot_target_checks.values()):
    failed = [name for name, passed in pilot_target_checks.items() if not passed]
    raise RuntimeError('Fold-0 target checks failed: ' + ', '.join(failed))

territory_edge_index_numpy = np.asarray(
    graph_schema['territory_edge_index'], dtype=np.int64
)
territory_edge_type_numpy = np.asarray(
    graph_schema['territory_edge_type'], dtype=np.int64
)
PILOT_RELATION_NAME = {
    1: 'template_contact_26n',
    2: 'contralateral_homolog',
    3: 'same_unilateral_parent',
}


def pilot_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, PILOT_PATCH_SIZE):
        if size <= patch:
            slices.append(slice(0, size))
            continue
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), size - patch)
        slices.append(slice(start, start + patch))
    return tuple(slices)


def pilot_extract_patch(array, slices, spatial_axes):
    source = [slice(None)] * array.ndim
    for axis, spatial_slice in zip(spatial_axes, slices):
        source[axis] = spatial_slice
    cropped = array[tuple(source)]
    return center_crop_or_pad(cropped, PILOT_PATCH_SIZE, spatial_axes)


region_cache = {}
region_cache_order = []
REGION_CACHE_LIMIT = 24


def pilot_load_region_map(subject_id):
    if subject_id in region_cache:
        return region_cache[subject_id]
    row = all_development_row_by_subject[subject_id]
    shape = pilot_shape_by_subject[subject_id]
    if not as_bool(row['trial8_atlas_enabled']):
        region = np.zeros(shape, dtype=np.uint8)
    else:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        atlas_image = nib.load(str(atlas_path))
        region = np.asanyarray(atlas_image.dataobj)
        if region.ndim == 4 and region.shape[-1] == 1:
            region = region[..., 0]
        region = np.rint(region).astype(np.uint8)
        if region.shape != shape:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region.min() < 0 or region.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    region_cache[subject_id] = region
    region_cache_order.append(subject_id)
    if len(region_cache_order) > REGION_CACHE_LIMIT:
        evicted = region_cache_order.pop(0)
        region_cache.pop(evicted, None)
    return region


def pilot_topology_candidates(support, region_map):
    lesion_labels = set(
        int(value) for value in np.unique(region_map[support])
        if 1 <= int(value) <= 30
    )
    relation_priority = {2: 0, 1: 1, 3: 2}
    relation_by_label = {}
    for edge_index in range(territory_edge_index_numpy.shape[1]):
        source = int(territory_edge_index_numpy[0, edge_index]) + 1
        destination = int(territory_edge_index_numpy[1, edge_index]) + 1
        relation = int(territory_edge_type_numpy[edge_index])
        if (
            source in lesion_labels
            and destination not in lesion_labels
            and relation in relation_priority
        ):
            if (
                destination not in relation_by_label
                or relation_priority[relation]
                < relation_priority[relation_by_label[destination]]
            ):
                relation_by_label[destination] = relation
    labels = sorted(
        relation_by_label,
        key=lambda label: (relation_priority[relation_by_label[label]], label),
    )
    return labels, relation_by_label


def pilot_sample_coordinates(mask, rng):
    flat = np.flatnonzero(mask)
    if len(flat) == 0:
        return np.empty((0, 3), dtype=np.int64)
    if len(flat) > MAXIMUM_NEGATIVE_CENTER_ATTEMPTS:
        flat = rng.choice(
            flat, size=MAXIMUM_NEGATIVE_CENTER_ATTEMPTS, replace=False
        )
    else:
        flat = flat.copy()
        rng.shuffle(flat)
    return np.column_stack(np.unravel_index(flat, mask.shape)).astype(np.int64)


def pilot_zero_support_center(exclusion, candidate, rng):
    for center in pilot_sample_coordinates(candidate, rng):
        slices = pilot_patch_slices(exclusion.shape, center)
        if not exclusion[slices].any():
            return center, slices
    return None, None


def pilot_positive_center(subject_id, rng, epoch):
    support = pilot_unpack_mask(
        pilot_mask_records[subject_id]['sampling_support']
    )
    if not support.any():
        raise RuntimeError(f'{subject_id}: positive schedule has no support.')
    if pilot_target_by_subject[subject_id]['supervision_mode'] == (
        'soft_only_subvoxel'
    ):
        labels, component_count = ndimage.label(
            support, structure=np.ones((3, 3, 3), dtype=np.uint8)
        )
        selected_component = 1 + ((epoch - 1) % component_count)
        coordinates = np.argwhere(labels == selected_component)
    else:
        coordinates = np.argwhere(support)
    seed = coordinates[int(rng.integers(0, len(coordinates)))].astype(np.int64)
    for _ in range(MAXIMUM_CENTER_RETRIES):
        jitter = np.asarray([
            int(rng.integers(-12, 13)),
            int(rng.integers(-12, 13)),
            int(rng.integers(-4, 5)),
        ])
        center = np.clip(seed + jitter, 0, np.asarray(support.shape) - 1)
        slices = pilot_patch_slices(support.shape, center)
        if support[slices].any():
            return center, slices
    slices = pilot_patch_slices(support.shape, seed)
    if not support[slices].any():
        raise RuntimeError(f'{subject_id}: failed to retain positive support.')
    return seed, slices


def pilot_balanced_positive_schedule(rng):
    schedule = []
    for lesion_bin in POSITIVE_BINS:
        pool = fold0_fit_df.loc[
            (fold0_fit_df['trial8_effective_lesion_volume_bin'] == lesion_bin)
            & fold0_fit_df['trial8_sampling_positive'].map(as_bool),
            'subject_id',
        ].astype(str).tolist()
        if not pool:
            raise RuntimeError(f'Fold-0 positive bin {lesion_bin} is empty.')
        target_count = max(MINIMUM_POSITIVE_PATCHES_PER_BIN, len(pool))
        selected = pool.copy()
        hard_pool = [
            subject_id for subject_id in pool
            if pilot_target_by_subject[subject_id]['supervision_mode']
            == 'hard_and_soft'
        ]
        rng.shuffle(hard_pool)
        if not hard_pool and len(selected) < target_count:
            raise RuntimeError(f'No hard-target repeats for {lesion_bin}.')
        repeat_index = 0
        while len(selected) < target_count:
            selected.append(hard_pool[repeat_index % len(hard_pool)])
            repeat_index += 1
        rng.shuffle(selected)
        schedule.extend(selected)
    rng.shuffle(schedule)
    return schedule


def pilot_build_epoch_schedule(epoch):
    rng = np.random.default_rng(PILOT_SEED + epoch * 1_000_003)
    positive_subjects = pilot_balanced_positive_schedule(rng)
    positive_count = len(positive_subjects)
    topology_count = int(math.ceil(positive_count / 4.0))
    brain_count = int(math.floor(positive_count / 4.0))
    topology_pool = fold0_fit_df.loc[
        fold0_fit_df['trial8_atlas_enabled'].map(as_bool)
        & fold0_fit_df['trial8_sampling_positive'].map(as_bool)
        & fold0_fit_df['trial8_supervision_mode'].eq('hard_and_soft'),
        'subject_id',
    ].astype(str).tolist()
    brain_pool = fold0_fit_df.loc[
        ~fold0_fit_df['trial8_supervision_mode'].eq('soft_only_subvoxel'),
        'subject_id',
    ].astype(str).tolist()
    entries = []
    for subject_id in positive_subjects:
        center, _ = pilot_positive_center(subject_id, rng, epoch)
        entries.append({
            'sample_type': 'positive',
            'subject_id': subject_id,
            'center': center,
            'topology_relation': '',
        })
    for _ in range(topology_count):
        selected = None
        start = int(rng.integers(0, len(topology_pool)))
        for attempt in range(len(topology_pool)):
            subject_id = topology_pool[(start + attempt) % len(topology_pool)]
            support = pilot_unpack_mask(
                pilot_mask_records[subject_id]['sampling_support']
            )
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            region = pilot_load_region_map(subject_id)
            labels, relation_by_label = pilot_topology_candidates(support, region)
            for relation in [2, 1, 3]:
                relation_labels = [
                    label for label in labels
                    if relation_by_label[label] == relation
                ]
                if not relation_labels:
                    continue
                center, _ = pilot_zero_support_center(
                    exclusion, np.isin(region, relation_labels) & crop_mask, rng
                )
                if center is not None:
                    selected = {
                        'sample_type': 'topology_negative',
                        'subject_id': subject_id,
                        'center': center,
                        'topology_relation': PILOT_RELATION_NAME[relation],
                    }
                    break
            if selected is not None:
                break
        if selected is None:
            raise RuntimeError('Could not construct a topology-negative patch.')
        entries.append(selected)
    for _ in range(brain_count):
        selected = None
        start = int(rng.integers(0, len(brain_pool)))
        for attempt in range(len(brain_pool)):
            subject_id = brain_pool[(start + attempt) % len(brain_pool)]
            exclusion = pilot_unpack_mask(
                pilot_mask_records[subject_id]['exclusion_support']
            )
            crop_mask = pilot_unpack_mask(
                pilot_mask_records[subject_id]['crop_mask']
            )
            center, _ = pilot_zero_support_center(exclusion, crop_mask, rng)
            if center is not None:
                selected = {
                    'sample_type': 'brain_negative',
                    'subject_id': subject_id,
                    'center': center,
                    'topology_relation': '',
                }
                break
        if selected is None:
            raise RuntimeError('Could not construct a brain-negative patch.')
        entries.append(selected)
    rng.shuffle(entries)
    return entries






graph_input_cache = {}
clinical_input_cache = {}


def pilot_load_graph_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in graph_input_cache:
        return graph_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_graph_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: graph sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_valid_source = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    territory_mean_local = np.asarray(
        fold0_graph_scaler['territory_mean'], dtype=np.float32
    )
    territory_std_local = np.asarray(
        fold0_graph_scaler['territory_std'], dtype=np.float32
    )
    vessel_indices = np.asarray(
        fold0_graph_scaler['vessel_source_indices'], dtype=np.int64
    )
    case_indices = np.asarray(
        fold0_graph_scaler['case_source_indices'], dtype=np.int64
    )
    vessel_feature_valid = (
        vessel_valid_source[:, vessel_indices] & vessel_presence[:, None]
    )
    territory_scaled = masked_scale(
        node_features,
        node_feature_valid & node_valid[:, None],
        territory_mean_local[None],
        territory_std_local[None],
    )
    vessel_scaled = masked_scale(
        vessel_source[:, vessel_indices],
        vessel_feature_valid,
        np.asarray(fold0_graph_scaler['vessel_mean'], dtype=np.float32)[None],
        np.asarray(fold0_graph_scaler['vessel_std'], dtype=np.float32)[None],
    )
    case_selected = case_source[case_indices].astype(np.float32)
    case_scaled = case_selected.copy()
    case_scale_mask_local = np.asarray(
        fold0_graph_scaler['case_scale_mask']
    ) > 0
    case_mean_local = np.asarray(
        fold0_graph_scaler['case_mean'], dtype=np.float32
    )
    case_std_local = np.asarray(
        fold0_graph_scaler['case_std'], dtype=np.float32
    )
    case_scaled[case_scale_mask_local] = np.clip(
        (
            case_scaled[case_scale_mask_local]
            - case_mean_local[case_scale_mask_local]
        ) / case_std_local[case_scale_mask_local],
        -SCALER_CLIP,
        SCALER_CLIP,
    )
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean',
        'minimum_lvo_distance_mm', 'lvo_overlap_fraction',
    ]
    occlusion_graph_values = []
    occlusion_graph_valid = []
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = vessel_feature_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_graph_values.append(float(
                vessel_scaled[valid_nodes, feature_index].mean()
            ))
            occlusion_graph_valid.append(1.0)
        else:
            occlusion_graph_values.append(0.0)
            occlusion_graph_valid.append(0.0)
    output = {
        'territory_features': territory_scaled.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': vessel_scaled.astype(np.float32),
        'vessel_feature_valid': vessel_feature_valid.astype(np.float32),
        'vessel_valid': vessel_presence.astype(bool),
        'case_features': case_scaled.astype(np.float32),
        'cow_prior_valid': bool(
            case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
        ),
        'registration_confidence': float(
            case_source[
                SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
            ]
        ),
        'occlusion_graph_values': np.asarray(
            occlusion_graph_values, dtype=np.float32
        ),
        'occlusion_graph_valid': np.asarray(
            occlusion_graph_valid, dtype=np.float32
        ),
    }
    if not all(
        np.isfinite(value).all()
        for value in output.values()
        if np.asarray(value).dtype.kind in {'f', 'c'}
    ):
        raise RuntimeError(f'{subject_id}: scaled graph input is non-finite.')
    graph_input_cache[subject_id] = output
    return output


def pilot_load_clinical_input(row):
    subject_id = str(row['subject_id'])
    if subject_id in clinical_input_cache:
        return clinical_input_cache[subject_id]
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(row['trial8_clinical_sidecar_sha256']):
        raise RuntimeError(f'{subject_id}: clinical sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        names = decode_strings(data['feature_names'])
    scaler_names = decode_strings(fold0_clinical_scaler['feature_names'])
    if names != scaler_names or values.shape != (22,) or valid.shape != (22,):
        raise RuntimeError(f'{subject_id}: clinical schema changed.')
    imputed = values.copy()
    imputation = np.asarray(
        fold0_clinical_scaler['imputation'], dtype=np.float32
    )
    imputed[~valid] = imputation[~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(fold0_clinical_scaler['scale_mask']) > 0
    clinical_mean = np.asarray(fold0_clinical_scaler['mean'], dtype=np.float32)
    clinical_std = np.asarray(fold0_clinical_scaler['std'], dtype=np.float32)
    clinical_clip = float(
        np.asarray(fold0_clinical_scaler['clip_value']).reshape(-1)[0]
    )
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - clinical_mean[scale_mask])
        / clinical_std[scale_mask],
        -clinical_clip,
        clinical_clip,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    output = {
        'clinical_input': token,
        'clinical_available': np.asarray(valid.any(), dtype=bool),
    }
    clinical_input_cache[subject_id] = output
    return output


def pilot_model_input_from_patches(
    row,
    image_patch,
    region_patch,
    cow_patch,
    lvo_patch,
    lvo_present,
    apply_graph_mask=False,
    rng=None,
):
    graph = {
        key: np.asarray(value).copy()
        for key, value in pilot_load_graph_input(row).items()
    }
    clinical = pilot_load_clinical_input(row)
    if apply_graph_mask and rng is not None:
        mask = (
            rng.random(graph['territory_features'].shape) < 0.05
        ) & (graph['territory_feature_valid'] > 0)
        graph['territory_features'][mask] = 0.0
        graph['territory_feature_valid'][mask] = 0.0
    input_channels = EXPECTED_INPUT_CHANNELS
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch,
        lvo_patch,
    ]).astype(np.float32)
    confidence = np.asarray([
        float(vascular_patch[2].mean()),
        float(graph['cow_prior_valid']),
        float(np.clip(graph['registration_confidence'], 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_values = np.concatenate([
        np.asarray([float(lvo_present), float(lvo_patch.max())], dtype=np.float32),
        graph['occlusion_graph_values'],
    ]).astype(np.float32)
    occlusion_valid = np.concatenate([
        np.asarray([1.0, float(lvo_present)], dtype=np.float32),
        graph['occlusion_graph_valid'],
    ]).astype(np.float32)
    return {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': graph['territory_features'].astype(np.float32),
        'territory_feature_valid': graph[
            'territory_feature_valid'
        ].astype(np.float32),
        'territory_valid': graph['territory_valid'].astype(bool),
        'territory_reliability': graph['territory_reliability'].astype(np.float32),
        'vessel_features': graph['vessel_features'].astype(np.float32),
        'vessel_feature_valid': graph['vessel_feature_valid'].astype(np.float32),
        'vessel_valid': graph['vessel_valid'].astype(bool),
        'case_features': graph['case_features'].astype(np.float32),
        'atlas_enabled': np.asarray(
            as_bool(row['trial8_atlas_enabled']), dtype=bool
        ),
        'vascular_image': vascular_patch,
        'confidence_features': confidence,
        'occlusion_values': occlusion_values,
        'occlusion_valid': occlusion_valid,
        'clinical_input': clinical['clinical_input'].astype(np.float32),
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': clinical['clinical_available'],
    }


def pilot_intensity_augment(image, rng):
    output = np.asarray(image, dtype=np.float32).copy()
    intensity_channels = min(13, output.shape[0])
    gains = rng.uniform(0.92, 1.08, size=(intensity_channels, 1, 1, 1))
    offsets = rng.uniform(-0.02, 0.02, size=(intensity_channels, 1, 1, 1))
    output[:intensity_channels] = output[:intensity_channels] * gains + offsets
    if rng.random() < 0.50:
        noise_std = float(rng.uniform(0.0, 0.015))
        output[:intensity_channels] += rng.normal(
            0.0, noise_std, size=output[:intensity_channels].shape
        ).astype(np.float32)
    if rng.random() < 0.40:
        gamma = float(rng.uniform(0.90, 1.10))
        nonnegative_end = min(9, intensity_channels)
        output[:nonnegative_end] = np.power(
            np.clip(output[:nonnegative_end], 0.0, 1.0), gamma
        )
    output[:min(9, intensity_channels)] = np.clip(
        output[:min(9, intensity_channels)], 0.0, 1.0
    )
    if intensity_channels > 9:
        output[9:intensity_channels] = np.clip(
            output[9:intensity_channels], -1.0, 1.0
        )
    return output.astype(np.float32)


class Trial8Fold0PatchDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.counts = {}
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)
        self.counts = {
            name: sum(entry['sample_type'] == name for entry in self.schedule)
            for name in ['positive', 'topology_negative', 'brain_negative']
        }

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        with np.load(npz_path, allow_pickle=False) as data:
            image = np.asarray(data['image'], dtype=np.float32)
            input_channels = decode_strings(data['input_channels'])
            target_binary = np.asarray(data['target_binary'], dtype=np.float32)
            target_soft = np.asarray(data['target_soft'], dtype=np.float32)
            cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
            lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
            lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
            hard_weight = pilot_scalar(data, 'binary_loss_weight')
        if input_channels != EXPECTED_INPUT_CHANNELS:
            raise RuntimeError(f'{subject_id}: image channel order changed.')
        slices = pilot_patch_slices(target_binary.shape, entry['center'])
        image_patch = pilot_extract_patch(image, slices, (1, 2, 3))
        binary_patch = pilot_extract_patch(target_binary, slices, (0, 1, 2))
        soft_patch = pilot_extract_patch(target_soft, slices, (0, 1, 2))
        cow_patch = pilot_extract_patch(cow, slices, (0, 1, 2))
        lvo_patch = pilot_extract_patch(lvo, slices, (0, 1, 2))
        region_patch = pilot_extract_patch(
            pilot_load_region_map(subject_id), slices, (0, 1, 2)
        )
        if entry['sample_type'] == 'positive' and not bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: positive patch lost target support.')
        if entry['sample_type'] != 'positive' and bool((soft_patch > 0).any()):
            raise RuntimeError(f'{subject_id}: negative patch contains target support.')
        image_patch = pilot_intensity_augment(image_patch, rng)
        model_inputs = pilot_model_input_from_patches(
            row,
            image_patch,
            region_patch,
            cow_patch,
            lvo_patch,
            lvo_present,
            apply_graph_mask=True,
            rng=rng,
        )
        model_inputs.update({
            'target_binary': binary_patch[None].astype(np.float32),
            'target_soft': soft_patch[None].astype(np.float32),
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'supervision_mode': str(
                pilot_target_by_subject[subject_id]['supervision_mode']
            ),
        })
        return model_inputs


MODEL_INPUT_KEYS = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available',
]


def pilot_to_device(cpu_batch):
    output = {}
    for key, value in cpu_batch.items():
        if torch.is_tensor(value):
            output[key] = value.to(device, non_blocking=True)
        else:
            output[key] = value
    return output


def pilot_spatial_augment(batch):
    batch_size = int(batch['image'].shape[0])
    if float(torch.rand(1).item()) >= 0.60:
        batch['spatially_augmented_fraction'] = torch.tensor(
            0.0, device=batch['image'].device
        )
        return batch
    positive = [str(value) == 'positive' for value in batch['sample_type']]
    batch_device = batch['image'].device
    for _ in range(4):
        angles = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * math.radians(8.0)
        scales = 0.92 + torch.rand(batch_size, device=batch_device) * 0.16
        cosine = torch.cos(angles) * scales
        sine = torch.sin(angles) * scales
        theta = torch.zeros(
            batch_size, 3, 4, device=batch_device, dtype=torch.float32
        )
        theta[:, 0, 0] = cosine
        theta[:, 0, 1] = -sine
        theta[:, 1, 0] = sine
        theta[:, 1, 1] = cosine
        theta[:, 2, 2] = scales
        theta[:, 0, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        theta[:, 1, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.05
        theta[:, 2, 3] = (
            torch.rand(batch_size, device=batch_device) * 2.0 - 1.0
        ) * 0.025
        grid = F.affine_grid(theta, batch['image'].shape, align_corners=False)

        transformed = dict(batch)
        transformed_image = torch.empty_like(batch['image'])
        transformed_image[:, :13] = F.grid_sample(
            batch['image'][:, :13], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed_image[:, 13:] = F.grid_sample(
            batch['image'][:, 13:], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['image'] = transformed_image
        transformed_vascular = torch.empty_like(batch['vascular_image'])
        continuous_indices = [0, 1, 3, 4]
        transformed_vascular[:, continuous_indices] = F.grid_sample(
            batch['vascular_image'][:, continuous_indices], grid,
            mode='bilinear', padding_mode='zeros', align_corners=False,
        )
        transformed_vascular[:, 2:3] = F.grid_sample(
            batch['vascular_image'][:, 2:3], grid,
            mode='nearest', padding_mode='zeros', align_corners=False,
        )
        transformed['vascular_image'] = transformed_vascular
        transformed['target_soft'] = F.grid_sample(
            batch['target_soft'], grid, mode='bilinear',
            padding_mode='zeros', align_corners=False,
        )
        transformed['target_binary'] = F.grid_sample(
            batch['target_binary'], grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )
        transformed['region_map'] = F.grid_sample(
            batch['region_map'][:, None].float(), grid, mode='nearest',
            padding_mode='zeros', align_corners=False,
        )[:, 0].long()
        positive_support_retained = all(
            (not is_positive) or bool(
                (transformed['target_soft'][batch_index] > 0).any().item()
            )
            for batch_index, is_positive in enumerate(positive)
        )
        if not positive_support_retained:
            continue
        transformed['confidence_features'] = batch[
            'confidence_features'
        ].clone()
        transformed['confidence_features'][:, 0] = transformed_vascular[
            :, 2
        ].float().mean(dim=(1, 2, 3))
        transformed['occlusion_values'] = batch['occlusion_values'].clone()
        transformed['occlusion_values'][:, 1] = transformed_vascular[
            :, 4
        ].float().amax(dim=(1, 2, 3))
        transformed['spatially_augmented_fraction'] = torch.tensor(
            1.0, device=batch_device
        )
        return transformed
    batch['spatially_augmented_fraction'] = torch.tensor(
        0.0, device=batch_device
    )
    return batch






def pilot_soft_dice_per_case(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_soft_focal_bce_per_case(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, raw.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_per_case(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    error_floor = 1e-6
    return (
        (1.0 - tversky).clamp_min(error_floor).pow(gamma)
        - float(error_floor ** gamma)
    )


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    dice = pilot_soft_dice_per_case(logits, soft_target)
    focal_bce = pilot_soft_focal_bce_per_case(logits, soft_target)
    hard_tversky = pilot_focal_tversky_per_case(logits, binary_target)
    hard_contribution = 0.20 * hard_tversky * hard_weight.reshape(-1)
    total = 0.50 * dice + 0.30 * focal_bce + hard_contribution
    return total.mean(), {
        'soft_dice_loss': dice.mean(),
        'soft_focal_bce_loss': focal_bce.mean(),
        'hard_focal_tversky_loss': hard_tversky.mean(),
        'hard_contribution': hard_contribution.mean(),
    }


def pilot_node_loss(output, batch):
    node_logits = output['territory_node_logits'].float()
    targets = torch.zeros_like(node_logits)
    valid = (
        batch['territory_valid'].bool()
        & batch['atlas_enabled'][:, None].bool()
        & (batch['hard_weight'][:, None] > 0)
    )
    for batch_index in range(node_logits.shape[0]):
        labels = batch['region_map'][batch_index]
        lesion = batch['target_binary'][batch_index, 0] > 0.5
        if bool(lesion.any().item()):
            present_labels = torch.unique(labels[lesion])
            present_labels = present_labels[
                (present_labels >= 1) & (present_labels <= 30)
            ]
            if len(present_labels) > 0:
                targets[batch_index, present_labels.long() - 1] = 1.0
    if not bool(valid.any().item()):
        return node_logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        node_logits,
        targets,
        reduction='none',
        pos_weight=torch.full(
            (30,), 4.0, device=node_logits.device, dtype=node_logits.dtype
        ),
    )
    return raw[valid].mean()


def pilot_compute_training_loss(study_model, cpu_batch):
    batch = pilot_to_device(cpu_batch)
    batch = pilot_spatial_augment(batch)
    output = study_model(**{key: batch[key] for key in MODEL_INPUT_KEYS})
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    auxiliary_eligible = (batch['hard_weight'] > 0).float()
    half_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], size=output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], size=output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'],
        size=output['auxiliary_quarter_logits'].shape[2:], mode='nearest',
    )
    half_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_half_logits'], half_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_half_logits'], half_binary
        ) * batch['hard_weight']
    )
    quarter_per_case = (
        0.50 * pilot_soft_dice_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.30 * pilot_soft_focal_bce_per_case(
            output['auxiliary_quarter_logits'], quarter_soft
        )
        + 0.20 * pilot_focal_tversky_per_case(
            output['auxiliary_quarter_logits'], quarter_binary
        ) * batch['hard_weight']
    )
    denominator = auxiliary_eligible.sum().clamp_min(1.0)
    half_loss = (half_per_case * auxiliary_eligible).sum() / denominator
    quarter_loss = (quarter_per_case * auxiliary_eligible).sum() / denominator
    node_loss = pilot_node_loss(output, batch)
    total = main_loss + 0.15 * half_loss + 0.10 * quarter_loss + 0.02 * node_loss
    parts = {
        'total_loss': total,
        'main_loss': main_loss,
        **main_parts,
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'atlas_retained_fraction': output[
            'effective_atlas_enabled'
        ].float().mean(),
        'vascular_retained_fraction': output[
            'effective_vascular_enabled'
        ].float().mean(),
        'occlusion_retained_fraction': output[
            'effective_occlusion_enabled'
        ].float().mean(),
        'clinical_retained_fraction': output[
            'effective_clinical_enabled'
        ].float().mean(),
        'spatially_augmented_fraction': batch[
            'spatially_augmented_fraction'
        ],
    }
    return total, parts


def pilot_sliding_starts(size, patch, overlap=0.5):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_extract_window(array, starts, spatial_axes):
    output_shape = list(array.shape)
    for axis, patch in zip(spatial_axes, PILOT_PATCH_SIZE):
        output_shape[axis] = patch
    output = np.zeros(output_shape, dtype=array.dtype)
    source = [slice(None)] * array.ndim
    destination = [slice(None)] * array.ndim
    source_spatial = []
    destination_spatial = []
    for axis, start, patch in zip(spatial_axes, starts, PILOT_PATCH_SIZE):
        stop = min(start + patch, array.shape[axis])
        copied = stop - start
        destination_start = max((patch - copied) // 2, 0)
        source[axis] = slice(start, stop)
        destination[axis] = slice(destination_start, destination_start + copied)
        source_spatial.append(slice(start, stop))
        destination_spatial.append(
            slice(destination_start, destination_start + copied)
        )
    output[tuple(destination)] = array[tuple(source)]
    return output, tuple(source_spatial), tuple(destination_spatial)


def pilot_gaussian_weight():
    coordinates = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in PILOT_PATCH_SIZE
    ]
    grids = np.meshgrid(*coordinates, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    weight = np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32)
    return np.maximum(weight, 0.05)


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


@torch.no_grad()
def pilot_full_volume_probability(study_model, row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: validation channel order changed.')
    region = pilot_load_region_map(subject_id)
    spatial_shape = image.shape[1:]
    probability_sum = np.zeros(spatial_shape, dtype=np.float32)
    weight_sum = np.zeros(spatial_shape, dtype=np.float32)
    window_records = []
    for x in pilot_sliding_starts(spatial_shape[0], PILOT_PATCH_SIZE[0]):
        for y in pilot_sliding_starts(spatial_shape[1], PILOT_PATCH_SIZE[1]):
            for z in pilot_sliding_starts(spatial_shape[2], PILOT_PATCH_SIZE[2]):
                window_records.append((x, y, z))
    study_model.eval()
    for start_index in range(0, len(window_records), 2):
        chunk = window_records[start_index:start_index + 2]
        numpy_inputs = []
        mappings = []
        for starts in chunk:
            image_patch, source_slices, destination_slices = pilot_extract_window(
                image, starts, (1, 2, 3)
            )
            region_patch, _, _ = pilot_extract_window(region, starts, (0, 1, 2))
            cow_patch, _, _ = pilot_extract_window(cow, starts, (0, 1, 2))
            lvo_patch, _, _ = pilot_extract_window(lvo, starts, (0, 1, 2))
            numpy_inputs.append(pilot_model_input_from_patches(
                row, image_patch, region_patch, cow_patch, lvo_patch,
                lvo_present, apply_graph_mask=False,
            ))
            mappings.append((source_slices, destination_slices))
        model_batch = {
            key: torch.from_numpy(np.stack([
                np.asarray(item[key]) for item in numpy_inputs
            ])).to(device, non_blocking=True)
            for key in MODEL_INPUT_KEYS
        }
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            output = study_model(
                **model_batch,
                force_atlas_dropout=False,
                force_vascular_dropout=False,
                force_occlusion_dropout=False,
                force_clinical_dropout=False,
            )
        probabilities = torch.sigmoid(output['logits'].float())[
            :, 0
        ].cpu().numpy()
        for local_index, probability_patch in enumerate(probabilities):
            source_slices, destination_slices = mappings[local_index]
            patch_probability = probability_patch[destination_slices]
            patch_weight = PILOT_WINDOW_WEIGHT[destination_slices]
            probability_sum[source_slices] += patch_probability * patch_weight
            weight_sum[source_slices] += patch_weight
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    if not np.isfinite(probability).all() or bool((weight_sum <= 0).any()):
        raise RuntimeError(f'{subject_id}: invalid sliding-window output.')
    return probability.astype(np.float32), target_binary, spacing


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap_pairs = np.column_stack([
        predicted_labels[target & prediction],
        target_labels[target & prediction],
    ])
    overlap_pairs = overlap_pairs[
        (overlap_pairs[:, 0] > 0) & (overlap_pairs[:, 1] > 0)
    ]
    if len(overlap_pairs) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap_pairs, axis=0, return_counts=True)
    order = np.argsort(-counts)
    used_predicted = set()
    used_target = set()
    matches = 0
    for pair_index in order:
        predicted_id, target_id = [int(value) for value in pairs[pair_index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


PILOT_THRESHOLDS = np.round(np.arange(0.10, 0.81, 0.05), 2)


@torch.no_grad()
def pilot_validate(study_model, epoch):
    case_cache = []
    for case_index, row in enumerate(heldout_rows, start=1):
        probability, target, spacing = pilot_full_volume_probability(
            study_model, row
        )
        case_cache.append({
            'row': row,
            'probability': probability.astype(np.float16),
            'target': target,
            'spacing': spacing,
        })
        print(
            f'  Held-out inference {case_index:02d}/{len(heldout_rows)}: '
            f'{row["subject_id"]}'
        )
    threshold_rows = []
    case_threshold_rows = []
    for threshold in PILOT_THRESHOLDS:
        positive_dice = []
        global_intersection = 0
        global_prediction = 0
        global_target = 0
        for case in case_cache:
            row = case['row']
            prediction = case['probability'] >= threshold
            target = case['target']
            intersection = int((prediction & target).sum())
            predicted_voxels = int(prediction.sum())
            target_voxels = int(target.sum())
            dice = (
                (2.0 * intersection) / (predicted_voxels + target_voxels)
                if predicted_voxels + target_voxels > 0
                else 1.0
            )
            hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
            if hard_metric_eligible and target_voxels > 0:
                positive_dice.append(dice)
                global_intersection += intersection
                global_prediction += predicted_voxels
                global_target += target_voxels
            case_threshold_rows.append({
                'epoch': epoch,
                'subject_id': str(row['subject_id']),
                'threshold': float(threshold),
                'hard_metric_eligible': hard_metric_eligible,
                'target_voxels': target_voxels,
                'predicted_voxels': predicted_voxels,
                'intersection_voxels': intersection,
                'dice': float(dice),
            })
        if not positive_dice:
            raise RuntimeError('Fold-0 held-out set has no positive hard targets.')
        global_dice = (
            2.0 * global_intersection
            / max(global_prediction + global_target, 1)
        )
        threshold_rows.append({
            'threshold': float(threshold),
            'positive_mean_dice': float(np.mean(positive_dice)),
            'positive_median_dice': float(np.median(positive_dice)),
            'positive_global_dice': float(global_dice),
            'positive_subjects': len(positive_dice),
        })
    threshold_df = pd.DataFrame(threshold_rows).sort_values(
        ['positive_mean_dice', 'positive_global_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected_threshold = float(threshold_df.iloc[0]['threshold'])
    selected_case_rows = []
    lesion_tp = lesion_fp = lesion_fn = 0
    total_intersection = total_prediction = total_target = 0
    positive_dice = []
    precisions = []
    recalls = []
    volume_errors = []
    empty_fp_volumes = []
    bin_dice = {name: [] for name in POSITIVE_BINS}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        prediction = probability >= selected_threshold
        target = case['target']
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0
            else 1.0
        )
        precision = intersection / max(predicted_voxels, 1)
        recall = intersection / max(target_voxels, 1)
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        predicted_volume_ml = predicted_voxels * voxel_volume_ml
        target_volume_ml = target_voxels * voxel_volume_ml
        hard_metric_eligible = as_bool(row['trial8_hard_metric_eligible'])
        true_positive_lesions = false_positive_lesions = false_negative_lesions = 0
        if hard_metric_eligible:
            (
                true_positive_lesions,
                false_positive_lesions,
                false_negative_lesions,
            ) = pilot_lesion_counts(prediction, target)
            lesion_tp += true_positive_lesions
            lesion_fp += false_positive_lesions
            lesion_fn += false_negative_lesions
            volume_errors.append(abs(predicted_volume_ml - target_volume_ml))
            if target_voxels > 0:
                positive_dice.append(dice)
                precisions.append(precision)
                recalls.append(recall)
                total_intersection += intersection
                total_prediction += predicted_voxels
                total_target += target_voxels
                lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
                if lesion_bin in bin_dice:
                    bin_dice[lesion_bin].append(dice)
            else:
                empty_fp_volumes.append(predicted_volume_ml)
        selected_case_rows.append({
            'epoch': epoch,
            'subject_id': str(row['subject_id']),
            'threshold': selected_threshold,
            'supervision_mode': str(row['trial8_supervision_mode']),
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': hard_metric_eligible,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'precision': float(precision),
            'recall': float(recall),
            'target_volume_ml': target_volume_ml,
            'predicted_volume_ml': predicted_volume_ml,
            'absolute_volume_error_ml': abs(
                predicted_volume_ml - target_volume_ml
            ),
            'lesion_true_positive': true_positive_lesions,
            'lesion_false_positive': false_positive_lesions,
            'lesion_false_negative': false_negative_lesions,
        })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    result = {
        'threshold': selected_threshold,
        'positive_mean_dice': float(np.mean(positive_dice)),
        'positive_median_dice': float(np.median(positive_dice)),
        'positive_global_dice': float(
            2.0 * total_intersection / max(total_prediction + total_target, 1)
        ),
        'mean_voxel_precision': float(np.mean(precisions)),
        'mean_voxel_recall': float(np.mean(recalls)),
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'mean_absolute_volume_error_ml': float(np.mean(volume_errors)),
        'mean_empty_false_positive_volume_ml': float(
            np.mean(empty_fp_volumes) if empty_fp_volumes else 0.0
        ),
        'positive_subjects': len(positive_dice),
        'hard_metric_subjects': int(sum(
            as_bool(case['row']['trial8_hard_metric_eligible'])
            for case in case_cache
        )),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{epoch:03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(threshold_df, epoch_root / 'threshold_summary.csv')
    atomic_write_csv(
        pd.DataFrame(case_threshold_rows), epoch_root / 'case_threshold_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(selected_case_rows), epoch_root / 'selected_case_metrics.csv'
    )
    atomic_write_json(result, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    return result






DIAGNOSTIC_VERSION = 'v2_2_fold0_failure_diagnostic_r1'
DIAGNOSTIC_CODE_REVISION = 'r1_branch_component_threshold_overfit_audit'
OFFICIAL_LESION_IOU_THRESHOLD = 0.20
BOOTSTRAP_REPLICATES = 5000
DIAGNOSTIC_THRESHOLDS = np.unique(np.concatenate([
    np.asarray([0.01, 0.02, 0.03, 0.04, 0.05, 0.075], dtype=np.float32),
    np.arange(0.10, 0.901, 0.025, dtype=np.float32),
])).astype(np.float32)
DIAGNOSTIC_MODES = [
    'full',
    'no_clinical',
    'no_occlusion',
    'no_vascular',
    'no_graph',
    'preserved_v1',
    'image_only',
]
SOURCE_CHECKPOINT_PATHS = {
    'v2': (
        PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
        / 'v1_fold0_v2_multimodal_ema_calibrated' / 'best_checkpoint.pt'
    ),
    'v2_1': (
        PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
        / 'v2_1_fold0_v2_multimodal_ema_calibrated' / 'best_checkpoint.pt'
    ),
}

DIAGNOSTIC_QA_ROOT = PILOT_ROOT / 'visual_qa'
DIAGNOSTIC_QA_ROOT.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_CONFIG_PATH = PILOT_ROOT / 'diagnostic_config.json'
DIAGNOSTIC_CHECKPOINT_AUDIT_PATH = PILOT_ROOT / 'source_checkpoint_audit.csv'
DIAGNOSTIC_MODE_SUMMARY_PATH = PILOT_ROOT / 'mode_summary.csv'
DIAGNOSTIC_CASE_METRICS_PATH = PILOT_ROOT / 'case_metrics.csv'
DIAGNOSTIC_THRESHOLD_PATH = PILOT_ROOT / 'threshold_sweep.csv'
DIAGNOSTIC_COMPONENT_PATH = PILOT_ROOT / 'target_component_audit.csv'
DIAGNOSTIC_HYSTERESIS_PATH = PILOT_ROOT / 'hysteresis_sweep.csv'
DIAGNOSTIC_GENERALIZATION_PATH = PILOT_ROOT / 'fit_heldout_generalization.csv'
DIAGNOSTIC_BOOTSTRAP_PATH = PILOT_ROOT / 'paired_bootstrap.csv'
DIAGNOSTIC_CORRELATION_PATH = PILOT_ROOT / 'quality_correlation_audit.csv'
DIAGNOSTIC_DECISION_PATH = PILOT_ROOT / 'diagnostic_decision_matrix.csv'
DIAGNOSTIC_SUMMARY_PATH = PILOT_ROOT / 'diagnostic_summary.json'
DIAGNOSTIC_GATE_PATH = PILOT_ROOT / 'diagnostic_gate.json'


def diagnostic_build_model():
    study_model = VascuFateNetV2(
        schema=graph_schema,
        image_channels=15,
        territory_feature_dim=65,
        vessel_feature_dim=15,
        case_feature_dim=8,
        graph_hidden=GRAPH_HIDDEN,
        graph_heads=GRAPH_HEADS,
        graph_layers=GRAPH_LAYERS,
        relation_bases=RELATION_BASES,
        graph_dropout=GRAPH_DROPOUT,
        atlas_dropout=ATLAS_DROPOUT_PROBABILITY,
        film_bound=FILM_BOUND,
        vascular_channels=VASCULAR_CHANNELS,
        clinical_input_dim=CLINICAL_INPUT_DIM,
        clinical_embed_dim=CLINICAL_EMBED_DIM,
        occlusion_value_dim=OCCLUSION_VALUE_DIM,
        occlusion_embed_dim=OCCLUSION_EMBED_DIM,
        confidence_dim=CONFIDENCE_DIM,
        fusion_bound=FUSION_BOUND,
    ).to(device)
    if parameter_count(study_model) != EXPECTED_V2_PARAMETERS:
        raise RuntimeError('Diagnostic model parameter count changed.')
    return study_model


def diagnostic_checkpoint_state(checkpoint):
    if 'ema_model_state_dict' not in checkpoint:
        raise RuntimeError('Source checkpoint does not contain an EMA state.')
    return checkpoint['ema_model_state_dict']


def diagnostic_checkpoint_audit():
    missing = [
        str(path) for path in SOURCE_CHECKPOINT_PATHS.values()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            'Completed source checkpoint(s) are missing:\n' + '\n'.join(missing)
        )
    reference_model = diagnostic_build_model().cpu()
    reference_state = reference_model.state_dict()
    del reference_model
    rows = []
    checkpoints = {}
    state_key_sets = []
    for source_name, checkpoint_path in SOURCE_CHECKPOINT_PATHS.items():
        checkpoint = torch.load(
            checkpoint_path, map_location='cpu', weights_only=False
        )
        state = diagnostic_checkpoint_state(checkpoint)
        state_key_sets.append(set(state))
        missing_keys = sorted(set(reference_state) - set(state))
        extra_keys = sorted(set(state) - set(reference_state))
        shape_errors = sorted(
            name for name in set(reference_state) & set(state)
            if tuple(reference_state[name].shape) != tuple(state[name].shape)
        )
        fixed_threshold = float(checkpoint.get('best_threshold', np.nan))
        row = {
            'source_model': source_name,
            'checkpoint_path': str(checkpoint_path),
            'checkpoint_sha256': file_sha256(checkpoint_path),
            'pilot_version': str(checkpoint.get('pilot_version', '')),
            'code_revision': str(checkpoint.get('code_revision', '')),
            'best_epoch': int(checkpoint.get('best_epoch', -1)),
            'best_mean_dice': float(checkpoint.get('best_mean_dice', np.nan)),
            'fixed_threshold': fixed_threshold,
            'parameters': int(checkpoint.get('parameters', -1)),
            'state_keys': len(state),
            'missing_state_keys': len(missing_keys),
            'extra_state_keys': len(extra_keys),
            'shape_errors': len(shape_errors),
            'locked_validation_targets_read': bool(
                checkpoint.get('locked_validation_targets_read', True)
            ),
            'test_targets_read': bool(
                checkpoint.get('test_targets_read', True)
            ),
            'outcome_workbooks_read': bool(
                checkpoint.get('outcome_workbooks_read', True)
            ),
        }
        checks = {
            'parameter_count_matches': row['parameters'] == EXPECTED_V2_PARAMETERS,
            'state_keys_match': not missing_keys and not extra_keys,
            'state_shapes_match': not shape_errors,
            'fixed_threshold_is_valid': 0.0 < fixed_threshold < 1.0,
            'locked_validation_targets_not_read': not row[
                'locked_validation_targets_read'
            ],
            'test_targets_not_read': not row['test_targets_read'],
            'outcome_workbooks_not_read': not row['outcome_workbooks_read'],
        }
        row['checkpoint_contract_passed'] = bool(all(checks.values()))
        row['failed_checks'] = ';'.join(
            name for name, passed in checks.items() if not passed
        )
        rows.append(row)
        checkpoints[source_name] = checkpoint
    if state_key_sets[0] != state_key_sets[1]:
        raise RuntimeError('V2 and V2.1 EMA state keys differ.')
    audit = pd.DataFrame(rows)
    atomic_write_csv(audit, DIAGNOSTIC_CHECKPOINT_AUDIT_PATH)
    if not bool(audit['checkpoint_contract_passed'].all()):
        raise RuntimeError('Source checkpoint contract failed; see checkpoint audit.')
    return checkpoints, audit


def diagnostic_zero_graph_inputs(model_batch):
    zero_keys = [
        'territory_features', 'territory_feature_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'case_features',
    ]
    false_keys = ['territory_valid', 'vessel_valid', 'atlas_enabled']
    model_batch['region_map'] = torch.zeros_like(model_batch['region_map'])
    for key in zero_keys:
        model_batch[key] = torch.zeros_like(model_batch[key])
    for key in false_keys:
        model_batch[key] = torch.zeros_like(model_batch[key], dtype=torch.bool)


def diagnostic_mode_controls(model_batch, mode):
    if mode not in DIAGNOSTIC_MODES:
        raise ValueError(f'Unknown diagnostic mode: {mode}')
    no_graph = mode in {'no_graph', 'image_only'}
    if no_graph:
        diagnostic_zero_graph_inputs(model_batch)
    return {
        'force_atlas_dropout': bool(no_graph),
        'force_vascular_dropout': mode in {
            'no_vascular', 'preserved_v1', 'image_only'
        },
        'force_occlusion_dropout': mode in {
            'no_occlusion', 'preserved_v1', 'image_only'
        },
        'force_clinical_dropout': mode in {
            'no_clinical', 'preserved_v1', 'image_only'
        },
    }


@torch.no_grad()
def diagnostic_full_volume_probability(study_model, row, mode):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target_binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: diagnostic channel order changed.')
    region = pilot_load_region_map(subject_id)
    spatial_shape = image.shape[1:]
    probability_sum = np.zeros(spatial_shape, dtype=np.float32)
    weight_sum = np.zeros(spatial_shape, dtype=np.float32)
    gate_sums = {
        'context_gate_mean_abs': 0.0,
        'quarter_gate_mean_abs': 0.0,
        'eighth_gate_mean_abs': 0.0,
        'bottleneck_gate_mean_abs': 0.0,
    }
    gate_examples = 0
    window_records = [
        (x, y, z)
        for x in pilot_sliding_starts(spatial_shape[0], PILOT_PATCH_SIZE[0])
        for y in pilot_sliding_starts(spatial_shape[1], PILOT_PATCH_SIZE[1])
        for z in pilot_sliding_starts(spatial_shape[2], PILOT_PATCH_SIZE[2])
    ]
    study_model.eval()
    for start_index in range(0, len(window_records), 2):
        chunk = window_records[start_index:start_index + 2]
        numpy_inputs = []
        mappings = []
        for starts in chunk:
            image_patch, source_slices, destination_slices = pilot_extract_window(
                image, starts, (1, 2, 3)
            )
            region_patch, _, _ = pilot_extract_window(region, starts, (0, 1, 2))
            cow_patch, _, _ = pilot_extract_window(cow, starts, (0, 1, 2))
            lvo_patch, _, _ = pilot_extract_window(lvo, starts, (0, 1, 2))
            numpy_inputs.append(pilot_model_input_from_patches(
                row, image_patch, region_patch, cow_patch, lvo_patch,
                lvo_present, apply_graph_mask=False,
            ))
            mappings.append((source_slices, destination_slices))
        model_batch = {
            key: torch.from_numpy(np.stack([
                np.asarray(item[key]) for item in numpy_inputs
            ])).to(device, non_blocking=True)
            for key in MODEL_INPUT_KEYS
        }
        controls = diagnostic_mode_controls(model_batch, mode)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == 'cuda',
        ):
            output = study_model(**model_batch, **controls)
        probabilities = torch.sigmoid(output['logits'].float())[:, 0].cpu().numpy()
        gate_map = {
            'context_gate_mean_abs': 'context_fusion_gate',
            'quarter_gate_mean_abs': 'vascular_quarter_gate',
            'eighth_gate_mean_abs': 'vascular_eighth_gate',
            'bottleneck_gate_mean_abs': 'vascular_bottleneck_gate',
        }
        for result_name, output_name in gate_map.items():
            gate_sums[result_name] += float(
                output[output_name].float().abs().mean().cpu()
            ) * len(chunk)
        gate_examples += len(chunk)
        for local_index, probability_patch in enumerate(probabilities):
            source_slices, destination_slices = mappings[local_index]
            patch_probability = probability_patch[destination_slices]
            patch_weight = PILOT_WINDOW_WEIGHT[destination_slices]
            probability_sum[source_slices] += patch_probability * patch_weight
            weight_sum[source_slices] += patch_weight
    probability = probability_sum / np.maximum(weight_sum, 1e-6)
    if not np.isfinite(probability).all() or bool((weight_sum <= 0).any()):
        raise RuntimeError(f'{subject_id}: invalid diagnostic output.')
    gate_means = {
        name: value / max(gate_examples, 1)
        for name, value in gate_sums.items()
    }
    return (
        probability.astype(np.float32), target_binary, spacing,
        region.astype(np.uint8), gate_means,
    )


def diagnostic_dice(prediction, target):
    intersection = int((prediction & target).sum())
    denominator = int(prediction.sum()) + int(target.sum())
    return 2.0 * intersection / denominator if denominator else 1.0


def diagnostic_official_component_match(
    prediction, target, iou_threshold=OFFICIAL_LESION_IOU_THRESHOLD
):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    target_sizes = np.bincount(
        target_labels.reshape(-1), minlength=target_count + 1
    )
    overlap_mask = (predicted_labels > 0) & (target_labels > 0)
    matches = []
    if bool(overlap_mask.any()):
        predicted_overlap = predicted_labels[overlap_mask].astype(np.int64)
        target_overlap = target_labels[overlap_mask].astype(np.int64)
        pair_base = int(target_count) + 1
        pair_codes = predicted_overlap * pair_base + target_overlap
        unique_codes, overlap_counts = np.unique(pair_codes, return_counts=True)
        candidate_predicted = np.unique(unique_codes // pair_base)
        candidate_target = np.unique(unique_codes % pair_base)
        predicted_lookup = {
            int(value): index for index, value in enumerate(candidate_predicted)
        }
        target_lookup = {
            int(value): index for index, value in enumerate(candidate_target)
        }
        iou_matrix = np.zeros(
            (len(candidate_predicted), len(candidate_target)), dtype=np.float64
        )
        for code, overlap in zip(unique_codes, overlap_counts):
            predicted_id = int(code // pair_base)
            target_id = int(code % pair_base)
            union = (
                int(predicted_sizes[predicted_id])
                + int(target_sizes[target_id]) - int(overlap)
            )
            iou_matrix[
                predicted_lookup[predicted_id], target_lookup[target_id]
            ] = float(overlap) / max(union, 1)
        row_indices, column_indices = linear_sum_assignment(-iou_matrix)
        for row_index, column_index in zip(row_indices, column_indices):
            iou = float(iou_matrix[row_index, column_index])
            if iou >= iou_threshold:
                matches.append({
                    'predicted_component': int(candidate_predicted[row_index]),
                    'target_component': int(candidate_target[column_index]),
                    'iou': iou,
                })
    return {
        'matches': matches,
        'true_positive': len(matches),
        'false_positive': int(predicted_count - len(matches)),
        'false_negative': int(target_count - len(matches)),
        'predicted_count': int(predicted_count),
        'target_count': int(target_count),
        'predicted_labels': predicted_labels,
        'target_labels': target_labels,
        'predicted_sizes': predicted_sizes,
        'target_sizes': target_sizes,
    }


def diagnostic_surface_metrics(prediction, target, spacing):
    if not bool(prediction.any()) or not bool(target.any()):
        return {
            'surface_dice_2mm': np.nan,
            'surface_dice_3mm': np.nan,
            'hd95_mm': np.nan,
            'assd_mm': np.nan,
        }
    structure = ndimage.generate_binary_structure(3, 1)
    predicted_surface = prediction & ~ndimage.binary_erosion(
        prediction, structure=structure, border_value=0
    )
    target_surface = target & ~ndimage.binary_erosion(
        target, structure=structure, border_value=0
    )
    distance_to_target = ndimage.distance_transform_edt(
        ~target_surface, sampling=tuple(float(value) for value in spacing)
    )
    distance_to_prediction = ndimage.distance_transform_edt(
        ~predicted_surface, sampling=tuple(float(value) for value in spacing)
    )
    predicted_to_target = distance_to_target[predicted_surface]
    target_to_predicted = distance_to_prediction[target_surface]
    combined = np.concatenate([predicted_to_target, target_to_predicted])
    denominator = max(len(predicted_to_target) + len(target_to_predicted), 1)
    return {
        'surface_dice_2mm': float(
            ((predicted_to_target <= 2.0).sum()
             + (target_to_predicted <= 2.0).sum()) / denominator
        ),
        'surface_dice_3mm': float(
            ((predicted_to_target <= 3.0).sum()
             + (target_to_predicted <= 3.0).sum()) / denominator
        ),
        'hd95_mm': float(np.percentile(combined, 95)),
        'assd_mm': float(combined.mean()),
    }


def diagnostic_evaluate_cache(case_cache, threshold, include_surfaces=False):
    rows = []
    totals = {
        'intersection': 0, 'prediction': 0, 'target': 0,
        'lesion_tp': 0, 'lesion_fp': 0, 'lesion_fn': 0,
    }
    positive_dice = []
    bin_dice = {name: [] for name in POSITIVE_BINS}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = diagnostic_dice(prediction, target)
        precision = intersection / max(predicted_voxels, 1)
        recall = intersection / max(target_voxels, 1)
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        component = diagnostic_official_component_match(prediction, target)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
        if hard_eligible:
            totals['lesion_tp'] += component['true_positive']
            totals['lesion_fp'] += component['false_positive']
            totals['lesion_fn'] += component['false_negative']
            if target_voxels > 0:
                positive_dice.append(dice)
                if lesion_bin in bin_dice:
                    bin_dice[lesion_bin].append(dice)
                totals['intersection'] += intersection
                totals['prediction'] += predicted_voxels
                totals['target'] += target_voxels
        surface = (
            diagnostic_surface_metrics(prediction, target, case['spacing'])
            if include_surfaces and hard_eligible and target_voxels > 0
            else {
                'surface_dice_2mm': np.nan,
                'surface_dice_3mm': np.nan,
                'hd95_mm': np.nan,
                'assd_mm': np.nan,
            }
        )
        rows.append({
            'subject_id': str(row['subject_id']),
            'lesion_volume_bin': lesion_bin,
            'hard_metric_eligible': hard_eligible,
            'threshold': float(threshold),
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'intersection_voxels': intersection,
            'dice': float(dice),
            'voxel_precision': float(precision),
            'voxel_recall': float(recall),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml': predicted_voxels * voxel_volume_ml,
            'absolute_volume_error_ml': abs(
                predicted_voxels - target_voxels
            ) * voxel_volume_ml,
            'target_components': component['target_count'],
            'predicted_components': component['predicted_count'],
            'lesion_true_positive_iou20': component['true_positive'],
            'lesion_false_positive_iou20': component['false_positive'],
            'lesion_false_negative_iou20': component['false_negative'],
            **surface,
            **case['gate_means'],
        })
    lesion_precision = totals['lesion_tp'] / max(
        totals['lesion_tp'] + totals['lesion_fp'], 1
    )
    lesion_recall = totals['lesion_tp'] / max(
        totals['lesion_tp'] + totals['lesion_fn'], 1
    )
    lesion_f1 = 2.0 * lesion_precision * lesion_recall / max(
        lesion_precision + lesion_recall, 1e-12
    )
    finite_bin_means = [
        float(np.mean(values)) for values in bin_dice.values() if values
    ]
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': float(np.mean(positive_dice)),
        'positive_median_dice': float(np.median(positive_dice)),
        'positive_global_dice': float(
            2.0 * totals['intersection']
            / max(totals['prediction'] + totals['target'], 1)
        ),
        'macro_lesion_bin_dice': float(np.mean(finite_bin_means)),
        'official_lesion_precision_iou20': float(lesion_precision),
        'official_lesion_recall_iou20': float(lesion_recall),
        'official_lesion_f1_iou20': float(lesion_f1),
        'positive_subjects': len(positive_dice),
        'target_components': int(totals['lesion_tp'] + totals['lesion_fn']),
        'predicted_components': int(totals['lesion_tp'] + totals['lesion_fp']),
    }
    for gate_name in [
        'context_gate_mean_abs', 'quarter_gate_mean_abs',
        'eighth_gate_mean_abs', 'bottleneck_gate_mean_abs',
    ]:
        summary[gate_name] = float(np.mean([
            row[gate_name] for row in rows
        ]))
    for name, values in bin_dice.items():
        summary[f'mean_dice_{name}'] = (
            float(np.mean(values)) if values else np.nan
        )
    summary['pareto_score'] = float(
        0.60 * summary['positive_mean_dice']
        + 0.20 * summary['macro_lesion_bin_dice']
        + 0.20 * summary['official_lesion_f1_iou20']
    )
    return summary, rows


def diagnostic_threshold_sweep(case_cache, fixed_threshold=None):
    
    
    
    
    thresholds = np.asarray(DIAGNOSTIC_THRESHOLDS, dtype=np.float64)
    bins = np.concatenate(([-np.inf], thresholds, [np.inf]))
    accumulators = [{
        'positive_dice': [],
        'bin_dice': {name: [] for name in POSITIVE_BINS},
        'intersection': 0,
        'prediction': 0,
        'target': 0,
    } for _ in thresholds]
    case_dice_by_threshold = {float(value): {} for value in thresholds}
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        target = case['target']
        all_histogram = np.histogram(probability, bins=bins)[0]
        target_histogram = np.histogram(probability[target], bins=bins)[0]
        predicted_counts = np.cumsum(all_histogram[::-1])[::-1][1:]
        intersection_counts = np.cumsum(target_histogram[::-1])[::-1][1:]
        target_voxels = int(target.sum())
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
        subject_id = str(row['subject_id'])
        for threshold_index, threshold in enumerate(thresholds):
            predicted_voxels = int(predicted_counts[threshold_index])
            intersection = int(intersection_counts[threshold_index])
            denominator = predicted_voxels + target_voxels
            dice = 2.0 * intersection / denominator if denominator else 1.0
            case_dice_by_threshold[float(threshold)][subject_id] = float(dice)
            if hard_eligible and target_voxels > 0:
                accumulator = accumulators[threshold_index]
                accumulator['positive_dice'].append(float(dice))
                if lesion_bin in accumulator['bin_dice']:
                    accumulator['bin_dice'][lesion_bin].append(float(dice))
                accumulator['intersection'] += intersection
                accumulator['prediction'] += predicted_voxels
                accumulator['target'] += target_voxels
    rows = []
    for threshold, accumulator in zip(thresholds, accumulators):
        bin_means = {
            name: float(np.mean(values)) if values else np.nan
            for name, values in accumulator['bin_dice'].items()
        }
        finite_bins = [value for value in bin_means.values() if math.isfinite(value)]
        rows.append({
            'threshold': float(threshold),
            'positive_mean_dice': float(np.mean(accumulator['positive_dice'])),
            'positive_median_dice': float(np.median(accumulator['positive_dice'])),
            'positive_global_dice': float(
                2.0 * accumulator['intersection']
                / max(accumulator['prediction'] + accumulator['target'], 1)
            ),
            'macro_lesion_bin_dice': float(np.mean(finite_bins)),
            'positive_subjects': len(accumulator['positive_dice']),
            'official_lesion_precision_iou20': np.nan,
            'official_lesion_recall_iou20': np.nan,
            'official_lesion_f1_iou20': np.nan,
            'pareto_score': np.nan,
            **{f'mean_dice_{name}': value for name, value in bin_means.items()},
        })
    table = pd.DataFrame(rows)
    calibrated = table.sort_values(
        ['positive_mean_dice', 'positive_global_dice', 'threshold'],
        ascending=[False, False, True],
    ).iloc[0].to_dict()
    top_dice_thresholds = table.nlargest(3, 'positive_mean_dice')[
        'threshold'
    ].tolist()
    anchor_thresholds = [0.10, 0.30, 0.50, 0.70]
    candidate_thresholds = set(top_dice_thresholds)
    candidate_thresholds.add(float(calibrated['threshold']))
    if fixed_threshold is not None:
        candidate_thresholds.add(float(fixed_threshold))
    for anchor in anchor_thresholds:
        candidate_thresholds.add(float(
            thresholds[np.argmin(np.abs(thresholds - anchor))]
        ))
    pareto_rows = []
    for threshold in sorted(candidate_thresholds):
        summary, _ = diagnostic_evaluate_cache(
            case_cache, threshold, include_surfaces=False
        )
        pareto_rows.append(summary)
        matching = np.isclose(table['threshold'].to_numpy(), threshold)
        for name in [
            'official_lesion_precision_iou20',
            'official_lesion_recall_iou20',
            'official_lesion_f1_iou20', 'target_components',
            'predicted_components', 'pareto_score',
        ]:
            table.loc[matching, name] = summary[name]
    pareto = pd.DataFrame(pareto_rows).sort_values(
        ['pareto_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).iloc[0].to_dict()
    oracle_rows = []
    for case in case_cache:
        subject_id = str(case['row']['subject_id'])
        candidates = [
            (float(threshold), case_dice_by_threshold[float(threshold)][subject_id])
            for threshold in DIAGNOSTIC_THRESHOLDS
        ]
        best_threshold, best_dice = sorted(
            candidates, key=lambda item: (-item[1], item[0])
        )[0]
        oracle_rows.append({
            'subject_id': subject_id,
            'oracle_threshold': best_threshold,
            'oracle_dice': float(best_dice),
        })
    return table, calibrated, pareto, pd.DataFrame(oracle_rows)


def diagnostic_hysteresis_prediction(
    probability, high_threshold, low_threshold, region_map=None,
    topology_gated=False,
):
    seeds = probability >= high_threshold
    low_mask = probability >= low_threshold
    if not bool(seeds.any()):
        return seeds
    if topology_gated and region_map is not None and bool((region_map > 0).any()):
        seed_labels = set(
            int(value) for value in np.unique(region_map[seeds]) if value > 0
        )
        allowed_labels = set(seed_labels)
        for edge_index in range(territory_edge_index_numpy.shape[1]):
            source = int(territory_edge_index_numpy[0, edge_index]) + 1
            destination = int(territory_edge_index_numpy[1, edge_index]) + 1
            if source in seed_labels or destination in seed_labels:
                allowed_labels.add(source)
                allowed_labels.add(destination)
        if allowed_labels:
            allowed = np.isin(region_map, sorted(allowed_labels))
            low_mask = low_mask & (allowed | seeds)
    return ndimage.binary_propagation(
        seeds, structure=np.ones((3, 3, 3), dtype=bool), mask=low_mask
    )


def diagnostic_hysteresis_sweep(case_cache, calibrated_threshold):
    high_values = sorted(set([
        float(calibrated_threshold), 0.30, 0.50, 0.70,
    ]))
    low_values = [0.01, 0.03, 0.05, 0.10, 0.20]
    rows = []
    for topology_gated in [False, True]:
        for high in high_values:
            for low in low_values:
                if low >= high:
                    continue
                dice_values = []
                tp = fp = fn = 0
                for case in case_cache:
                    prediction = diagnostic_hysteresis_prediction(
                        case['probability'].astype(np.float32), high, low,
                        case['region'], topology_gated=topology_gated,
                    )
                    target = case['target']
                    if as_bool(case['row']['trial8_hard_metric_eligible']) and bool(
                        target.any()
                    ):
                        dice_values.append(diagnostic_dice(prediction, target))
                    match = diagnostic_official_component_match(prediction, target)
                    tp += match['true_positive']
                    fp += match['false_positive']
                    fn += match['false_negative']
                precision = tp / max(tp + fp, 1)
                recall = tp / max(tp + fn, 1)
                rows.append({
                    'method': (
                        'topology_gated_hysteresis'
                        if topology_gated else 'spatial_hysteresis'
                    ),
                    'high_threshold': high,
                    'low_threshold': low,
                    'positive_mean_dice': float(np.mean(dice_values)),
                    'official_lesion_precision_iou20': float(precision),
                    'official_lesion_recall_iou20': float(recall),
                    'official_lesion_f1_iou20': float(
                        2.0 * precision * recall
                        / max(precision + recall, 1e-12)
                    ),
                })
    return pd.DataFrame(rows)


def diagnostic_target_component_rows(
    source_model, case_cache, selected_threshold
):
    rows = []
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    for case in case_cache:
        row = case['row']
        subject_id = str(row['subject_id'])
        probability = case['probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= selected_threshold
        selected_match = diagnostic_official_component_match(prediction, target)
        matched_by_target = {
            item['target_component']: item for item in selected_match['matches']
        }
        target_labels = selected_match['target_labels']
        target_count = selected_match['target_count']
        if bool(prediction.any()):
            distance_to_prediction = ndimage.distance_transform_edt(
                ~prediction,
                sampling=tuple(float(value) for value in case['spacing']),
            )
        else:
            distance_to_prediction = None
        oracle_by_target = {
            target_id: {'iou': 0.0, 'threshold': np.nan}
            for target_id in range(1, target_count + 1)
        }
        for threshold in DIAGNOSTIC_THRESHOLDS:
            candidate = probability >= float(threshold)
            candidate_match = diagnostic_official_component_match(
                candidate, target, iou_threshold=0.0
            )
            for item in candidate_match['matches']:
                target_id = item['target_component']
                if item['iou'] > oracle_by_target[target_id]['iou']:
                    oracle_by_target[target_id] = {
                        'iou': float(item['iou']),
                        'threshold': float(threshold),
                    }
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        for target_id in range(1, target_count + 1):
            mask = target_labels == target_id
            values = probability[mask]
            oracle = oracle_by_target[target_id]
            matched = matched_by_target.get(target_id)
            nearest_distance = (
                float(distance_to_prediction[mask].min())
                if distance_to_prediction is not None else np.nan
            )
            if matched is not None:
                failure_class = 'detected_iou20'
            elif oracle['iou'] >= OFFICIAL_LESION_IOU_THRESHOLD:
                failure_class = 'threshold_mismatch'
            elif float(values.max()) >= max(0.05, selected_threshold * 0.50):
                failure_class = 'topology_or_fragmentation'
            elif math.isfinite(nearest_distance) and nearest_distance <= 5.0:
                failure_class = 'near_miss_within_5mm'
            elif float(values.max()) >= 0.03:
                failure_class = 'weak_localization'
            else:
                failure_class = 'representation_failure'
            rows.append({
                'source_model': source_model,
                'subject_id': subject_id,
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'target_component': target_id,
                'component_voxels': int(mask.sum()),
                'component_volume_ml': float(mask.sum() * voxel_volume_ml),
                'selected_threshold': float(selected_threshold),
                'selected_iou': float(matched['iou']) if matched else 0.0,
                'oracle_iou': float(oracle['iou']),
                'oracle_threshold': oracle['threshold'],
                'probability_maximum': float(values.max()),
                'probability_mean': float(values.mean()),
                'probability_p95': float(np.percentile(values, 95)),
                'nearest_selected_prediction_mm': nearest_distance,
                'failure_class': failure_class,
            })
    return rows


def diagnostic_visual_qa(
    source_model, row, probability, target, threshold, output_path
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)[0]
    coordinates = np.argwhere(target)
    if len(coordinates):
        center = np.rint(coordinates.mean(axis=0)).astype(int)
    else:
        center = np.asarray(np.unravel_index(np.argmax(probability), probability.shape))
    prediction = probability >= threshold
    fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
    plane_names = ['sagittal', 'coronal', 'axial']
    for axis, plane_name in enumerate(plane_names):
        index = int(center[axis])
        image_plane = np.take(image, index, axis=axis).T
        target_plane = np.take(target, index, axis=axis).T
        probability_plane = np.take(probability, index, axis=axis).T
        prediction_plane = np.take(prediction, index, axis=axis).T
        axes[0, axis].imshow(image_plane, cmap='gray', origin='lower')
        if bool(target_plane.any()):
            axes[0, axis].contour(
                target_plane, levels=[0.5], colors='red', linewidths=1.0
            )
        axes[0, axis].set_title(f'{plane_name} {index}: target red')
        axes[1, axis].imshow(
            probability_plane, cmap='magma', origin='lower', vmin=0, vmax=1
        )
        if bool(target_plane.any()):
            axes[1, axis].contour(
                target_plane, levels=[0.5], colors='cyan', linewidths=1.0
            )
        if bool(prediction_plane.any()):
            axes[1, axis].contour(
                prediction_plane, levels=[0.5], colors='lime', linewidths=0.8
            )
        axes[1, axis].set_title(
            f'probability; target cyan, prediction lime @ {threshold:.3f}'
        )
        for row_index in range(2):
            axes[row_index, axis].axis('off')
    fig.suptitle(
        f'{source_model} {subject_id} '
        f'({row["trial8_effective_lesion_volume_bin"]})'
    )
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def diagnostic_bin_matched_fit_rows():
    selected = []
    heldout_positive = fold0_heldout_df[
        fold0_heldout_df['trial8_hard_metric_eligible'].map(as_bool)
        & fold0_heldout_df['trial8_sampling_positive'].map(as_bool)
    ]
    for lesion_bin, count in heldout_positive[
        'trial8_effective_lesion_volume_bin'
    ].value_counts().items():
        pool = fold0_fit_df[
            fold0_fit_df['trial8_hard_metric_eligible'].map(as_bool)
            & fold0_fit_df['trial8_sampling_positive'].map(as_bool)
            & fold0_fit_df['trial8_effective_lesion_volume_bin'].eq(lesion_bin)
        ].copy()
        pool['_selection_key'] = pool['subject_id'].astype(str).map(
            lambda value: hashlib.sha256(
                f'{PILOT_SEED}|fit_match|{lesion_bin}|{value}'.encode('utf-8')
            ).hexdigest()
        )
        selected.extend(
            pool.sort_values('_selection_key').head(int(count)).drop(
                columns=['_selection_key']
            ).to_dict('records')
        )
    return sorted(selected, key=lambda row: str(row['subject_id']))


def diagnostic_bootstrap_difference(
    left_values, right_values, seed, replicates=BOOTSTRAP_REPLICATES
):
    left = np.asarray(left_values, dtype=np.float64)
    right = np.asarray(right_values, dtype=np.float64)
    if left.shape != right.shape or left.ndim != 1 or len(left) == 0:
        raise ValueError('Paired bootstrap inputs are invalid.')
    difference = left - right
    rng = np.random.default_rng(seed)
    sampled = np.empty(replicates, dtype=np.float64)
    for index in range(replicates):
        selection = rng.integers(0, len(difference), size=len(difference))
        sampled[index] = difference[selection].mean()
    return {
        'subjects': len(difference),
        'mean_difference': float(difference.mean()),
        'ci95_lower': float(np.percentile(sampled, 2.5)),
        'ci95_upper': float(np.percentile(sampled, 97.5)),
        'probability_difference_positive': float((sampled > 0).mean()),
    }


def diagnostic_quality_features(row):
    subject_id = str(row['subject_id'])
    graph = pilot_load_graph_input(row)
    clinical = pilot_load_clinical_input(row)
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        perfusion_valid_fraction = float(
            image[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')].mean()
        )
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    return {
        'subject_id': subject_id,
        'atlas_enabled': float(as_bool(row['trial8_atlas_enabled'])),
        'registration_confidence': float(graph['registration_confidence']),
        'valid_territory_nodes': int(np.asarray(graph['territory_valid']).sum()),
        'valid_vessel_nodes': int(np.asarray(graph['vessel_valid']).sum()),
        'perfusion_valid_fraction': perfusion_valid_fraction,
        'lvo_present': float(lvo_present),
        'clinical_missing_features': int(
            np.asarray(clinical['clinical_input'])[22:].sum()
        ),
    }


def run_trial8_v22_failure_diagnostic():
    checkpoints, checkpoint_audit = diagnostic_checkpoint_audit()
    checkpoint_hashes = {
        row['source_model']: row['checkpoint_sha256']
        for row in checkpoint_audit.to_dict('records')
    }
    if DIAGNOSTIC_GATE_PATH.exists():
        existing = json.loads(DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8'))
        if (
            existing.get('diagnostic_passed') is True
            and existing.get('source_checkpoint_sha256') == checkpoint_hashes
            and existing.get('code_revision') == DIAGNOSTIC_CODE_REVISION
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 FAILURE DIAGNOSTIC ALREADY COMPLETE')
            print('=' * 80)
            print(f'Diagnostic gate: {DIAGNOSTIC_GATE_PATH}')
            print('No source checkpoint or diagnostic output was rewritten.')
            return

    diagnostic_config = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': DIAGNOSTIC_VERSION,
        'code_revision': DIAGNOSTIC_CODE_REVISION,
        'source_checkpoint_paths': {
            name: str(path) for name, path in SOURCE_CHECKPOINT_PATHS.items()
        },
        'source_checkpoint_sha256': checkpoint_hashes,
        'modes': DIAGNOSTIC_MODES,
        'thresholds': DIAGNOSTIC_THRESHOLDS.tolist(),
        'official_lesion_iou_threshold': OFFICIAL_LESION_IOU_THRESHOLD,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'fit_subjects': len(fold0_fit_df),
        'heldout_subjects': len(fold0_heldout_df),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'training_performed': False,
        'model_training_authorized': False,
    }
    atomic_write_json(diagnostic_config, DIAGNOSTIC_CONFIG_PATH)

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 READ-ONLY FAILURE DIAGNOSTIC START')
    print('=' * 80)
    print(f'Device:                             {device}')
    print(f'Fold-0 fit / held-out:              {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
    print(f'Source models:                      {list(SOURCE_CHECKPOINT_PATHS)}')
    print(f'Ablation modes:                     {DIAGNOSTIC_MODES}')
    print(f'Threshold candidates:               {len(DIAGNOSTIC_THRESHOLDS)}')
    print('Official lesion match:              one-to-one IoU >= 0.20')
    print('Locked validation/test targets:     sealed')
    print('Training/optimizer steps:            0')

    all_mode_summaries = []
    all_case_rows = []
    all_threshold_rows = []
    all_component_rows = []
    all_hysteresis_rows = []
    heldout_full_thresholds = {}
    heldout_full_case_rows = {}
    all_loaded_models_frozen = True
    quality_rows = [
        diagnostic_quality_features(row) for row in heldout_rows
    ]
    quality_df = pd.DataFrame(quality_rows)

    for source_index, source_name in enumerate(['v2', 'v2_1']):
        checkpoint = checkpoints[source_name]
        study_model = diagnostic_build_model()
        study_model.load_state_dict(
            diagnostic_checkpoint_state(checkpoint), strict=True
        )
        study_model.eval().requires_grad_(False)
        all_loaded_models_frozen = all_loaded_models_frozen and not any(
            parameter.requires_grad for parameter in study_model.parameters()
        )
        fixed_threshold = float(checkpoint['best_threshold'])
        for mode_index, mode in enumerate(DIAGNOSTIC_MODES, start=1):
            print(
                f'\n{source_name} mode {mode_index}/{len(DIAGNOSTIC_MODES)}: '
                f'{mode}'
            )
            case_cache = []
            for case_index, row in enumerate(heldout_rows, start=1):
                probability, target, spacing, region, gate_means = (
                    diagnostic_full_volume_probability(study_model, row, mode)
                )
                case_cache.append({
                    'row': row,
                    'probability': probability.astype(np.float16),
                    'target': target,
                    'spacing': spacing,
                    'region': region,
                    'gate_means': gate_means,
                })
                print(
                    f'  {case_index:02d}/{len(heldout_rows)} '
                    f'{row["subject_id"]}'
                )
            threshold_df, calibrated, pareto, oracle_df = (
                diagnostic_threshold_sweep(case_cache, fixed_threshold)
            )
            threshold_df.insert(0, 'mode', mode)
            threshold_df.insert(0, 'source_model', source_name)
            all_threshold_rows.extend(threshold_df.to_dict('records'))

            evaluation_specs = [
                ('checkpoint_fixed', fixed_threshold, False),
                ('mode_recalibrated', float(calibrated['threshold']), mode == 'full'),
                ('pareto_recalibrated', float(pareto['threshold']), False),
            ]
            for policy, threshold, include_surfaces in evaluation_specs:
                summary, case_rows = diagnostic_evaluate_cache(
                    case_cache, threshold, include_surfaces=include_surfaces
                )
                summary.update({
                    'source_model': source_name,
                    'mode': mode,
                    'evaluation_policy': policy,
                    'checkpoint_fixed_threshold': fixed_threshold,
                })
                all_mode_summaries.append(summary)
                oracle_lookup = oracle_df.set_index('subject_id').to_dict('index')
                for case_row in case_rows:
                    case_row.update({
                        'source_model': source_name,
                        'mode': mode,
                        'evaluation_policy': policy,
                        'oracle_threshold': oracle_lookup[
                            case_row['subject_id']
                        ]['oracle_threshold'],
                        'oracle_dice': oracle_lookup[
                            case_row['subject_id']
                        ]['oracle_dice'],
                        'oracle_minus_selected_dice': (
                            oracle_lookup[case_row['subject_id']]['oracle_dice']
                            - case_row['dice']
                        ),
                    })
                all_case_rows.extend(case_rows)

            if mode == 'full':
                calibrated_threshold = float(calibrated['threshold'])
                heldout_full_thresholds[source_name] = calibrated_threshold
                full_rows = [
                    row for row in all_case_rows
                    if row['source_model'] == source_name
                    and row['mode'] == 'full'
                    and row['evaluation_policy'] == 'mode_recalibrated'
                ]
                heldout_full_case_rows[source_name] = full_rows
                component_rows = diagnostic_target_component_rows(
                    source_name, case_cache, calibrated_threshold
                )
                all_component_rows.extend(component_rows)
                hysteresis_df = diagnostic_hysteresis_sweep(
                    case_cache, calibrated_threshold
                )
                hysteresis_df.insert(0, 'source_model', source_name)
                all_hysteresis_rows.extend(hysteresis_df.to_dict('records'))
                for case in case_cache:
                    lesion_bin = str(
                        case['row']['trial8_effective_lesion_volume_bin']
                    )
                    if lesion_bin in {'tiny', 'small'} and bool(case['target'].any()):
                        qa_path = (
                            DIAGNOSTIC_QA_ROOT
                            / f'{source_name}_{case["row"]["subject_id"]}_qa.png'
                        )
                        diagnostic_visual_qa(
                            source_name, case['row'],
                            case['probability'].astype(np.float32),
                            case['target'], calibrated_threshold, qa_path,
                        )
            del case_cache
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        del study_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    mode_summary_df = pd.DataFrame(all_mode_summaries)
    case_metrics_df = pd.DataFrame(all_case_rows)
    threshold_sweep_df = pd.DataFrame(all_threshold_rows)
    component_df = pd.DataFrame(all_component_rows)
    hysteresis_df = pd.DataFrame(all_hysteresis_rows)
    atomic_write_csv(mode_summary_df, DIAGNOSTIC_MODE_SUMMARY_PATH)
    atomic_write_csv(case_metrics_df, DIAGNOSTIC_CASE_METRICS_PATH)
    atomic_write_csv(threshold_sweep_df, DIAGNOSTIC_THRESHOLD_PATH)
    atomic_write_csv(component_df, DIAGNOSTIC_COMPONENT_PATH)
    atomic_write_csv(hysteresis_df, DIAGNOSTIC_HYSTERESIS_PATH)

    fit_subset_rows = diagnostic_bin_matched_fit_rows()
    expected_matched_fit_subjects = int((
        fold0_heldout_df['trial8_hard_metric_eligible'].map(as_bool)
        & fold0_heldout_df['trial8_sampling_positive'].map(as_bool)
    ).sum())
    if len(fit_subset_rows) != expected_matched_fit_subjects:
        raise RuntimeError(
            'Could not construct an exactly bin-matched fit subset: '
            f'{len(fit_subset_rows)} vs {expected_matched_fit_subjects}.'
        )
    fit_subjects = {str(row['subject_id']) for row in fit_subset_rows}
    heldout_subjects = {str(row['subject_id']) for row in heldout_rows}
    fit_bin_counts = pd.Series([
        str(row['trial8_effective_lesion_volume_bin'])
        for row in fit_subset_rows
    ]).value_counts().to_dict()
    heldout_bin_counts = fold0_heldout_df.loc[
        fold0_heldout_df['trial8_hard_metric_eligible'].map(as_bool)
        & fold0_heldout_df['trial8_sampling_positive'].map(as_bool),
        'trial8_effective_lesion_volume_bin',
    ].astype(str).value_counts().to_dict()
    if fit_subjects & heldout_subjects:
        raise RuntimeError('Bin-matched fit and held-out sets overlap.')
    generalization_rows = []
    for source_name in ['v2', 'v2_1']:
        checkpoint = checkpoints[source_name]
        study_model = diagnostic_build_model()
        study_model.load_state_dict(
            diagnostic_checkpoint_state(checkpoint), strict=True
        )
        study_model.eval().requires_grad_(False)
        all_loaded_models_frozen = all_loaded_models_frozen and not any(
            parameter.requires_grad for parameter in study_model.parameters()
        )
        fit_cache = []
        print(f'\n{source_name}: bin-matched fit-subset inference...')
        for case_index, row in enumerate(fit_subset_rows, start=1):
            probability, target, spacing, region, gate_means = (
                diagnostic_full_volume_probability(study_model, row, 'full')
            )
            fit_cache.append({
                'row': row,
                'probability': probability.astype(np.float16),
                'target': target,
                'spacing': spacing,
                'region': region,
                'gate_means': gate_means,
            })
            print(
                f'  {case_index:02d}/{len(fit_subset_rows)} '
                f'{row["subject_id"]}'
            )
        fit_summary, fit_cases = diagnostic_evaluate_cache(
            fit_cache, heldout_full_thresholds[source_name],
            include_surfaces=False,
        )
        heldout_cases = heldout_full_case_rows[source_name]
        heldout_positive = [
            row['dice'] for row in heldout_cases
            if row['hard_metric_eligible'] and row['target_voxels'] > 0
        ]
        for partition, rows, mean_dice in [
            ('bin_matched_fit', fit_cases, fit_summary['positive_mean_dice']),
            ('heldout', heldout_cases, float(np.mean(heldout_positive))),
        ]:
            generalization_rows.append({
                'source_model': source_name,
                'partition': partition,
                'subjects': len(rows),
                'threshold_from_heldout_full': heldout_full_thresholds[source_name],
                'positive_mean_dice': mean_dice,
            })
            for lesion_bin in POSITIVE_BINS:
                values = [
                    item['dice'] for item in rows
                    if item['lesion_volume_bin'] == lesion_bin
                    and item['hard_metric_eligible']
                    and item['target_voxels'] > 0
                ]
                generalization_rows[-1][f'mean_dice_{lesion_bin}'] = (
                    float(np.mean(values)) if values else np.nan
                )
        del fit_cache, study_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    generalization_df = pd.DataFrame(generalization_rows)
    atomic_write_csv(generalization_df, DIAGNOSTIC_GENERALIZATION_PATH)

    bootstrap_rows = []
    calibrated_cases = case_metrics_df[
        case_metrics_df['evaluation_policy'].eq('mode_recalibrated')
    ]
    for source_index, source_name in enumerate(['v2', 'v2_1']):
        for policy_index, evaluation_policy in enumerate([
            'checkpoint_fixed', 'mode_recalibrated'
        ]):
            source = case_metrics_df[
                case_metrics_df['source_model'].eq(source_name)
                & case_metrics_df['evaluation_policy'].eq(evaluation_policy)
            ]
            full = source[source['mode'].eq('full')].set_index('subject_id')
            for mode_index, mode in enumerate(DIAGNOSTIC_MODES[1:], start=1):
                ablated = source[source['mode'].eq(mode)].set_index('subject_id')
                common = sorted(set(full.index) & set(ablated.index))
                result = diagnostic_bootstrap_difference(
                    full.loc[common, 'dice'].to_numpy(),
                    ablated.loc[common, 'dice'].to_numpy(),
                    seed=(
                        PILOT_SEED + policy_index * 10_000
                        + source_index * 100 + mode_index
                    ),
                )
                bootstrap_rows.append({
                    'comparison': f'{source_name}:full_minus_{mode}',
                    'source_model': source_name,
                    'evaluation_policy': evaluation_policy,
                    'left_mode': 'full',
                    'right_mode': mode,
                    'subset': 'all_heldout',
                    **result,
                })
                small_common = [
                    subject_id for subject_id in common
                    if str(full.loc[subject_id, 'lesion_volume_bin']) in {
                        'tiny', 'small'
                    }
                ]
                if small_common:
                    result = diagnostic_bootstrap_difference(
                        full.loc[small_common, 'dice'].to_numpy(),
                        ablated.loc[small_common, 'dice'].to_numpy(),
                        seed=(
                            PILOT_SEED + 1000 + policy_index * 10_000
                            + source_index * 100 + mode_index
                        ),
                    )
                    bootstrap_rows.append({
                        'comparison': f'{source_name}:full_minus_{mode}',
                        'source_model': source_name,
                        'evaluation_policy': evaluation_policy,
                        'left_mode': 'full',
                        'right_mode': mode,
                        'subset': 'tiny_and_small',
                        **result,
                    })
    v2_full = calibrated_cases[
        calibrated_cases['source_model'].eq('v2')
        & calibrated_cases['mode'].eq('full')
    ].set_index('subject_id')
    v21_full = calibrated_cases[
        calibrated_cases['source_model'].eq('v2_1')
        & calibrated_cases['mode'].eq('full')
    ].set_index('subject_id')
    common = sorted(set(v2_full.index) & set(v21_full.index))
    bootstrap_rows.append({
        'comparison': 'v2_1_full_minus_v2_full',
        'source_model': 'cross_model',
        'evaluation_policy': 'mode_recalibrated',
        'left_mode': 'v2_1_full',
        'right_mode': 'v2_full',
        'subset': 'all_heldout',
        **diagnostic_bootstrap_difference(
            v21_full.loc[common, 'dice'].to_numpy(),
            v2_full.loc[common, 'dice'].to_numpy(),
            seed=PILOT_SEED + 9000,
        ),
    })
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    atomic_write_csv(bootstrap_df, DIAGNOSTIC_BOOTSTRAP_PATH)

    correlation_rows = []
    for source_name in ['v2', 'v2_1']:
        full = calibrated_cases[
            calibrated_cases['source_model'].eq(source_name)
            & calibrated_cases['mode'].eq('full')
        ].merge(quality_df, on='subject_id', how='left')
        candidate_features = [
            'target_volume_ml', 'target_components', 'atlas_enabled',
            'registration_confidence', 'valid_territory_nodes',
            'valid_vessel_nodes', 'perfusion_valid_fraction', 'lvo_present',
            'clinical_missing_features',
        ]
        for feature_name in candidate_features:
            for outcome_name in [
                'dice', 'absolute_volume_error_ml',
                'oracle_minus_selected_dice',
            ]:
                valid = (
                    np.isfinite(full[feature_name])
                    & np.isfinite(full[outcome_name])
                )
                if (
                    int(valid.sum()) >= 3
                    and full.loc[valid, feature_name].nunique() > 1
                    and full.loc[valid, outcome_name].nunique() > 1
                ):
                    rho, p_value = spearmanr(
                        full.loc[valid, feature_name],
                        full.loc[valid, outcome_name],
                    )
                else:
                    rho, p_value = np.nan, np.nan
                correlation_rows.append({
                    'source_model': source_name,
                    'feature': feature_name,
                    'outcome': outcome_name,
                    'subjects': int(valid.sum()),
                    'spearman_rho': (
                        float(rho) if math.isfinite(rho) else np.nan
                    ),
                    'two_sided_p_value_descriptive_only': (
                        float(p_value) if math.isfinite(p_value) else np.nan
                    ),
                    'confirmatory_inference': False,
                })
    correlation_df = pd.DataFrame(correlation_rows)
    atomic_write_csv(correlation_df, DIAGNOSTIC_CORRELATION_PATH)

    decision_rows = []
    for source_name in ['v2', 'v2_1']:
        full_summary = mode_summary_df[
            mode_summary_df['source_model'].eq(source_name)
            & mode_summary_df['mode'].eq('full')
            & mode_summary_df['evaluation_policy'].eq('mode_recalibrated')
        ].iloc[0]
        full_cases = calibrated_cases[
            calibrated_cases['source_model'].eq(source_name)
            & calibrated_cases['mode'].eq('full')
        ]
        component_source = component_df[
            component_df['source_model'].eq(source_name)
        ]
        oracle_gap = float(full_cases['oracle_minus_selected_dice'].mean())
        best_hysteresis = hysteresis_df[
            hysteresis_df['source_model'].eq(source_name)
        ].sort_values('positive_mean_dice', ascending=False).iloc[0]
        hysteresis_gain = float(
            best_hysteresis['positive_mean_dice']
            - full_summary['positive_mean_dice']
        )
        representation_failure_fraction = float(
            component_source['failure_class'].eq(
                'representation_failure'
            ).mean()
        ) if len(component_source) else np.nan
        fit_mean = float(generalization_df[
            generalization_df['source_model'].eq(source_name)
            & generalization_df['partition'].eq('bin_matched_fit')
        ]['positive_mean_dice'].iloc[0])
        heldout_mean = float(full_summary['positive_mean_dice'])
        overfit_gap = fit_mean - heldout_mean
        criteria = [
            (
                'threshold_or_postprocessing_limited',
                oracle_gap >= 0.03 or hysteresis_gain >= 0.015,
                max(oracle_gap, hysteresis_gain),
                'oracle gap >=0.03 or hysteresis gain >=0.015',
                'retain dense calibration and evaluate graph-guided hysteresis',
            ),
            (
                'representation_limited',
                representation_failure_fraction >= 0.30,
                representation_failure_fraction,
                'at least 30% of target components have maximum probability <0.03',
                'add a high-resolution lesion-candidate head and component loss',
            ),
            (
                'component_localization_limited',
                float(full_summary['official_lesion_recall_iou20']) < 0.30,
                float(full_summary['official_lesion_recall_iou20']),
                'official IoU20 lesion recall <0.30',
                'prioritize component-balanced sampling and directed vascular context',
            ),
            (
                'generalization_limited',
                overfit_gap >= 0.08,
                overfit_gap,
                'bin-matched fit minus held-out mean Dice >=0.08',
                'increase regularization and reduce conditioning capacity before expansion',
            ),
        ]
        for criterion, triggered, observed, rule, action in criteria:
            decision_rows.append({
                'source_model': source_name,
                'criterion': criterion,
                'triggered': bool(triggered),
                'observed_value': observed,
                'predeclared_rule': rule,
                'recommended_action_if_triggered': action,
            })
    for record in bootstrap_df[
        bootstrap_df['subset'].eq('all_heldout')
        & bootstrap_df['source_model'].isin(['v2', 'v2_1'])
        & bootstrap_df['evaluation_policy'].eq('checkpoint_fixed')
    ].to_dict('records'):
        if record['ci95_lower'] > 0.005:
            interpretation = 'branch_supported'
        elif record['ci95_upper'] < -0.005:
            interpretation = 'branch_harmful_or_misapplied'
        else:
            interpretation = 'inconclusive'
        decision_rows.append({
            'source_model': record['source_model'],
            'criterion': record['comparison'],
            'triggered': interpretation != 'inconclusive',
            'observed_value': record['mean_difference'],
            'predeclared_rule': (
                'paired bootstrap CI entirely above +0.005 supports branch; '
                'entirely below -0.005 indicates harm'
            ),
            'recommended_action_if_triggered': interpretation,
        })
    decision_rows.append({
        'source_model': 'policy',
        'criterion': 'post_treatment_routing',
        'triggered': False,
        'observed_value': np.nan,
        'predeclared_rule': (
            'primary prediction remains pre-treatment; post-treatment technique '
            'and reperfusion outcomes are excluded'
        ),
        'recommended_action_if_triggered': (
            'defer treatment-conditioned routing to a separately labeled '
            'retrospective secondary study'
        ),
    })
    decision_rows.append({
        'source_model': 'policy',
        'criterion': 'venous_topology_input',
        'triggered': False,
        'observed_value': np.nan,
        'predeclared_rule': (
            'the frozen sidecars contain arterial territory/vessel topology, '
            'not a verified venous graph'
        ),
        'recommended_action_if_triggered': (
            'do not label arterial features as venous; require a separately '
            'registered, target-free venous source and QA gate before use'
        ),
    })
    decision_df = pd.DataFrame(decision_rows)
    atomic_write_csv(decision_df, DIAGNOSTIC_DECISION_PATH)

    expected_mode_evaluations = (
        len(SOURCE_CHECKPOINT_PATHS) * len(DIAGNOSTIC_MODES) * 3
    )
    diagnostic_checks = {
        'both_source_checkpoints_verified': len(checkpoint_audit) == 2
        and bool(checkpoint_audit['checkpoint_contract_passed'].all()),
        'all_fourteen_model_mode_inferences_completed': (
            mode_summary_df[['source_model', 'mode']].drop_duplicates().shape[0]
            == len(SOURCE_CHECKPOINT_PATHS) * len(DIAGNOSTIC_MODES)
        ),
        'all_evaluation_policies_completed': len(mode_summary_df)
        == expected_mode_evaluations,
        'all_heldout_subjects_evaluated_per_mode': bool(
            case_metrics_df.groupby([
                'source_model', 'mode', 'evaluation_policy'
            ])['subject_id'].nunique().eq(len(heldout_rows)).all()
        ),
        'official_iou20_matching_used': math.isclose(
            OFFICIAL_LESION_IOU_THRESHOLD, 0.20
        ),
        'dense_low_threshold_grid_used': float(DIAGNOSTIC_THRESHOLDS.min())
        <= 0.01,
        'component_failure_audit_nonempty': len(component_df) > 0,
        'spatial_and_topology_hysteresis_tested': set(
            hysteresis_df['method']
        ) == {'spatial_hysteresis', 'topology_gated_hysteresis'},
        'fit_subset_is_disjoint_from_heldout': not bool(
            fit_subjects & heldout_subjects
        ),
        'fit_subset_count_matches_heldout_positive_count': (
            len(fit_subset_rows) == expected_matched_fit_subjects
        ),
        'fit_subset_lesion_bins_exactly_match_heldout': (
            fit_bin_counts == heldout_bin_counts
        ),
        'paired_bootstrap_completed': len(bootstrap_df) >= 13,
        'quality_correlations_completed': len(correlation_df) == 54,
        'decision_matrix_written': len(decision_df) > 0,
        'model_parameters_frozen_during_diagnostic': all_loaded_models_frozen,
        'no_optimizer_steps_performed': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'outcome_workbooks_not_read': True,
        'post_treatment_fields_not_used': True,
        'source_checkpoints_not_modified': all(
            file_sha256(path) == checkpoint_hashes[name]
            for name, path in SOURCE_CHECKPOINT_PATHS.items()
        ),
        'diagnostic_output_is_isolated': str(PILOT_ROOT).endswith(PILOT_VERSION),
        'model_training_remains_locked': True,
    }
    diagnostic_passed = bool(all(diagnostic_checks.values()))

    concise_full = mode_summary_df[
        mode_summary_df['mode'].eq('full')
        & mode_summary_df['evaluation_policy'].eq('mode_recalibrated')
    ][[
        'source_model', 'threshold', 'positive_mean_dice',
        'positive_median_dice', 'positive_global_dice',
        'macro_lesion_bin_dice', 'official_lesion_f1_iou20',
        'official_lesion_recall_iou20', 'mean_dice_tiny',
        'mean_dice_small', 'mean_dice_medium', 'mean_dice_large',
        'mean_dice_very_large',
    ]]
    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': DIAGNOSTIC_VERSION,
        'code_revision': DIAGNOSTIC_CODE_REVISION,
        'diagnostic_passed': diagnostic_passed,
        'checks': diagnostic_checks,
        'source_checkpoint_sha256': checkpoint_hashes,
        'full_model_results': concise_full.to_dict('records'),
        'official_lesion_iou_threshold': OFFICIAL_LESION_IOU_THRESHOLD,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'fit_subset_subjects': len(fit_subset_rows),
        'heldout_subjects': len(heldout_rows),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'training_performed': False,
        'study_trainable_parameters': 0,
        'model_training_authorized': False,
        'next_architecture_authorized': False,
        'decision_matrix_path': str(DIAGNOSTIC_DECISION_PATH),
    }
    atomic_write_json(summary, DIAGNOSTIC_SUMMARY_PATH)
    atomic_write_json({
        **summary,
        'diagnostic_output_root': str(PILOT_ROOT),
        'mode_summary_path': str(DIAGNOSTIC_MODE_SUMMARY_PATH),
        'case_metrics_path': str(DIAGNOSTIC_CASE_METRICS_PATH),
        'component_audit_path': str(DIAGNOSTIC_COMPONENT_PATH),
        'hysteresis_path': str(DIAGNOSTIC_HYSTERESIS_PATH),
        'bootstrap_path': str(DIAGNOSTIC_BOOTSTRAP_PATH),
        'correlation_path': str(DIAGNOSTIC_CORRELATION_PATH),
        'visual_qa_root': str(DIAGNOSTIC_QA_ROOT),
    }, DIAGNOSTIC_GATE_PATH)

    print('\nFull-model recalibrated results:')
    display(concise_full)
    print('\nPredeclared decision matrix:')
    display(decision_df)
    print('\nDiagnostic checks:')
    display(pd.DataFrame([
        {'check': name, 'passed': passed}
        for name, passed in diagnostic_checks.items()
    ]))
    if not diagnostic_passed:
        failed = [
            name for name, passed in diagnostic_checks.items() if not passed
        ]
        raise RuntimeError(
            'TRIAL8 V2.2 FAILURE DIAGNOSTIC FAILED. Do not revise or train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 FAILURE DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Source checkpoints:                 {len(SOURCE_CHECKPOINT_PATHS)}')
    print(f'Model/ablation combinations:        {len(SOURCE_CHECKPOINT_PATHS) * len(DIAGNOSTIC_MODES)}')
    print(f'Held-out subjects per combination:  {len(heldout_rows)}')
    print(f'Bin-matched fit subjects:           {len(fit_subset_rows)}')
    print(f'Official lesion matching IoU:       {OFFICIAL_LESION_IOU_THRESHOLD:.2f}')
    print(f'Bootstrap replicates:               {BOOTSTRAP_REPLICATES}')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Outcome/post-treatment data read:   False')
    print('Training/optimizer steps:           0')
    print('Study trainable parameters:         0 (diagnostic only)')
    print('Model training authorized:          False')
    print('Next architecture authorized:       False (review evidence first)')
    print(f'Diagnostic root:                    {PILOT_ROOT}')
    print(f'Decision matrix:                    {DIAGNOSTIC_DECISION_PATH}')
    print(f'Diagnostic gate:                    {DIAGNOSTIC_GATE_PATH}')
    print('\nNext: check the full output tables and tiny/small visual-QA panels.')


run_trial8_v22_failure_diagnostic()





_INERT_ORIGINAL_PILOT_TRAINING_PROVENANCE = 'MAX_EPOCHS = int(fivefold_policy[\'maximum_epochs\'])\nPATIENCE_VALIDATIONS = int(\n    fivefold_policy[\'early_stopping_patience_validations\']\n)\nEARLY_STOPPING_MINIMUM_EPOCH = int(\n    fivefold_policy[\'early_stopping_minimum_epoch\']\n)\nEARLY_STOPPING_MINIMUM_DELTA = float(\n    fivefold_policy[\'early_stopping_minimum_delta\']\n)\nBASE_LEARNING_RATE = float(fivefold_policy[\'learning_rate\'])\nWEIGHT_DECAY = float(fivefold_policy[\'weight_decay\'])\nACCUMULATION_STEPS = int(fivefold_policy[\'gradient_accumulation_steps\'])\nGRADIENT_CLIP_NORM = float(fivefold_policy[\'gradient_clip_norm\'])\nEMA_DECAY = float(fivefold_policy[\'ema_decay\'])\nBATCH_SIZE = int(fivefold_policy[\'batch_size\'])\nGATE_LEARNING_RATE_MULTIPLIER = 5.0\nNUM_WORKERS = 0\n\npilot_policy_checks = {\n    \'patch_size_matches\': tuple(fivefold_policy[\'patch_size\']) == PILOT_PATCH_SIZE,\n    \'batch_size_is_one\': BATCH_SIZE == 1,\n    \'accumulation_steps_are_four\': ACCUMULATION_STEPS == 4,\n    \'maximum_epochs_are_60\': MAX_EPOCHS == 60,\n    \'validation_every_epoch\': int(\n        fivefold_policy[\'validation_interval_epochs\']\n    ) == 1,\n    \'metrics_printed_every_epoch\': fivefold_policy[\n        \'metrics_printed_every_epoch\'\n    ] is True,\n    \'early_stopping_minimum_epoch_is_20\': (\n        EARLY_STOPPING_MINIMUM_EPOCH == 20\n    ),\n    \'early_stopping_patience_is_15\': PATIENCE_VALIDATIONS == 15,\n    \'early_stopping_delta_is_0p002\': math.isclose(\n        EARLY_STOPPING_MINIMUM_DELTA, 0.002\n    ),\n    \'volume_case_weights_disabled\': fivefold_policy[\n        \'volume_case_loss_weighting\'\n    ] is False,\n    \'top_k_ohem_disabled\': fivefold_policy[\'loss\'][\'top_k_ohem\'] is False,\n    \'left_right_reflection_disabled\': fivefold_policy[\n        \'left_right_reflection\'\n    ] is False,\n    \'checkpoint_metric_matches\': fivefold_policy[\n        \'checkpoint_metric\'\n    ] == \'heldout_positive_mean_dice\',\n    \'locked_validation_policy_preserved\': \'do not read\' in fivefold_policy[\n        \'locked_validation_policy\'\n    ],\n    \'test_policy_preserved\': \'sealed\' in fivefold_policy[\'test_policy\'],\n}\nif not all(pilot_policy_checks.values()):\n    failed = [name for name, passed in pilot_policy_checks.items() if not passed]\n    raise RuntimeError(\'Frozen pilot policy changed: \' + \', \'.join(failed))\n\nset_determinism(PILOT_SEED)\ntraining_dataset = Trial8Fold0PatchDataset()\nepoch1_schedule = training_dataset.schedule\nschedule_rows = [{\n    \'sample_index\': index,\n    \'sample_type\': entry[\'sample_type\'],\n    \'subject_id\': entry[\'subject_id\'],\n    \'center_x\': int(entry[\'center\'][0]),\n    \'center_y\': int(entry[\'center\'][1]),\n    \'center_z\': int(entry[\'center\'][2]),\n    \'topology_relation\': entry[\'topology_relation\'],\n} for index, entry in enumerate(epoch1_schedule)]\natomic_write_csv(pd.DataFrame(schedule_rows), PILOT_SCHEDULE_AUDIT_PATH)\n\npilot_schedule_checks = {\n    \'epoch_has_168_patches\': len(epoch1_schedule) == 168,\n    \'positive_patches_are_112\': training_dataset.counts[\'positive\'] == 112,\n    \'topology_negative_patches_are_28\': (\n        training_dataset.counts[\'topology_negative\'] == 28\n    ),\n    \'brain_negative_patches_are_28\': (\n        training_dataset.counts[\'brain_negative\'] == 28\n    ),\n    \'positive_fraction_is_two_thirds\': math.isclose(\n        training_dataset.counts[\'positive\'] / len(epoch1_schedule),\n        2.0 / 3.0,\n    ),\n    \'all_fit_positive_subjects_covered\': set(\n        fold0_fit_df.loc[\n            fold0_fit_df[\'trial8_sampling_positive\'].map(as_bool), \'subject_id\'\n        ].astype(str)\n    ).issubset({\n        entry[\'subject_id\'] for entry in epoch1_schedule\n        if entry[\'sample_type\'] == \'positive\'\n    }),\n}\nif not all(pilot_schedule_checks.values()):\n    failed = [name for name, passed in pilot_schedule_checks.items() if not passed]\n    raise RuntimeError(\'Fold-0 epoch schedule changed: \' + \', \'.join(failed))\n\ntraining_model = VascuFateNetV2(\n    schema=graph_schema,\n    image_channels=15,\n    territory_feature_dim=65,\n    vessel_feature_dim=15,\n    case_feature_dim=8,\n    graph_hidden=GRAPH_HIDDEN,\n    graph_heads=GRAPH_HEADS,\n    graph_layers=GRAPH_LAYERS,\n    relation_bases=RELATION_BASES,\n    graph_dropout=GRAPH_DROPOUT,\n    atlas_dropout=ATLAS_DROPOUT_PROBABILITY,\n    film_bound=FILM_BOUND,\n    vascular_channels=VASCULAR_CHANNELS,\n    clinical_input_dim=CLINICAL_INPUT_DIM,\n    clinical_embed_dim=CLINICAL_EMBED_DIM,\n    occlusion_value_dim=OCCLUSION_VALUE_DIM,\n    occlusion_embed_dim=OCCLUSION_EMBED_DIM,\n    confidence_dim=CONFIDENCE_DIM,\n    fusion_bound=FUSION_BOUND,\n).to(device)\nif parameter_count(training_model) != EXPECTED_V2_PARAMETERS:\n    raise RuntimeError(\'Fold-0 pilot model parameter count changed.\')\n\nsmoke_loader = torch.utils.data.DataLoader(\n    training_dataset,\n    batch_size=1,\n    shuffle=False,\n    num_workers=0,\n    pin_memory=device.type == \'cuda\',\n)\nsmoke_cpu_batch = next(iter(smoke_loader))\ntraining_model.train()\nwith torch.autocast(\n    device_type=device.type,\n    dtype=torch.bfloat16,\n    enabled=device.type == \'cuda\',\n):\n    pilot_smoke_loss, pilot_smoke_parts = pilot_compute_training_loss(\n        training_model, smoke_cpu_batch\n    )\nif not bool(torch.isfinite(pilot_smoke_loss).item()):\n    raise RuntimeError(\'Fold-0 pilot smoke loss is non-finite.\')\npilot_smoke_loss.backward()\npilot_smoke_gradient_finite = all(\n    parameter.grad is None or bool(torch.isfinite(parameter.grad).all().item())\n    for parameter in training_model.parameters()\n)\npilot_smoke_gradient_nonzero = any(\n    parameter.grad is not None and bool((parameter.grad != 0).any().item())\n    for parameter in training_model.parameters()\n)\ntraining_model.zero_grad(set_to_none=True)\n\npilot_start_checks = {\n    **pilot_upstream_checks,\n    **pilot_manifest_checks,\n    **pilot_scaler_contract_checks,\n    **pilot_target_checks,\n    **pilot_policy_checks,\n    **pilot_schedule_checks,\n    \'model_parameter_count_matches\': parameter_count(training_model)\n    == EXPECTED_V2_PARAMETERS,\n    \'smoke_loss_finite\': bool(torch.isfinite(pilot_smoke_loss).item()),\n    \'smoke_gradients_finite\': pilot_smoke_gradient_finite,\n    \'smoke_has_nonzero_gradient\': pilot_smoke_gradient_nonzero,\n    \'target_access_is_development_train_only\': True,\n    \'locked_validation_targets_not_read\': True,\n    \'test_targets_not_read\': True,\n    \'outcome_workbooks_not_read\': True,\n    \'expert_routing_disabled\': True,\n    \'direct_graph_to_logit_residual_disabled\': True,\n}\npilot_training_authorized = bool(all(pilot_start_checks.values()))\natomic_write_json({\n    \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n    \'pilot_version\': PILOT_VERSION,\n    \'code_revision\': PILOT_CODE_REVISION,\n    \'pilot_training_authorized\': pilot_training_authorized,\n    \'checks\': pilot_start_checks,\n    \'fold\': HELDOUT_FOLD,\n    \'fit_subjects\': len(fold0_fit_df),\n    \'heldout_subjects\': len(fold0_heldout_df),\n    \'locked_validation_targets_read\': False,\n    \'test_targets_read\': False,\n    \'outcome_workbooks_read\': False,\n    \'full_fivefold_training_authorized\': False,\n}, PILOT_START_GATE_PATH)\nif not pilot_training_authorized:\n    failed = [name for name, passed in pilot_start_checks.items() if not passed]\n    raise RuntimeError(\n        \'TRIAL8 V2 FOLD-0 PILOT START GATE FAILED. Do not train. Failed: \'\n        + \', \'.join(failed)\n    )\n\nema_model = copy.deepcopy(training_model).to(device).eval()\nema_model.requires_grad_(False)\ngate_parameters = []\nstandard_parameters = []\nfor name, parameter in training_model.named_parameters():\n    if \'.gate.network.2.\' in name:\n        gate_parameters.append(parameter)\n    else:\n        standard_parameters.append(parameter)\nif not gate_parameters:\n    raise RuntimeError(\'Zero-initialized fusion-gate parameters were not found.\')\ntraining_optimizer = torch.optim.AdamW([\n    {\n        \'params\': standard_parameters,\n        \'lr\': BASE_LEARNING_RATE,\n        \'weight_decay\': WEIGHT_DECAY,\n        \'group_name\': \'standard\',\n    },\n    {\n        \'params\': gate_parameters,\n        \'lr\': BASE_LEARNING_RATE * GATE_LEARNING_RATE_MULTIPLIER,\n        \'weight_decay\': 0.0,\n        \'group_name\': \'zero_init_fusion_gates\',\n    },\n])\nupdates_per_epoch = math.ceil(len(training_dataset) / ACCUMULATION_STEPS)\ntotal_updates = MAX_EPOCHS * updates_per_epoch\nwarmup_updates = 3 * updates_per_epoch\n\n\ndef pilot_lr_lambda(update_index):\n    completed = update_index + 1\n    if completed <= warmup_updates:\n        return completed / max(warmup_updates, 1)\n    progress = (completed - warmup_updates) / max(\n        total_updates - warmup_updates, 1\n    )\n    return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))\n\n\ntraining_scheduler = torch.optim.lr_scheduler.LambdaLR(\n    training_optimizer, lr_lambda=pilot_lr_lambda\n)\n\n\ndef pilot_update_ema(ema, study, maximum_decay, update_step):\n    decay = min(maximum_decay, (1.0 + update_step) / (10.0 + update_step))\n    with torch.no_grad():\n        for ema_parameter, parameter in zip(\n            ema.parameters(), study.parameters()\n        ):\n            ema_parameter.mul_(decay).add_(parameter, alpha=1.0 - decay)\n        for ema_buffer, buffer in zip(ema.buffers(), study.buffers()):\n            ema_buffer.copy_(buffer)\n\n\ndef pilot_epoch_loader(epoch):\n    training_dataset.set_epoch(epoch)\n    generator = torch.Generator()\n    generator.manual_seed(PILOT_SEED + epoch * 97_409)\n    return torch.utils.data.DataLoader(\n        training_dataset,\n        batch_size=BATCH_SIZE,\n        shuffle=False,\n        num_workers=NUM_WORKERS,\n        pin_memory=device.type == \'cuda\',\n        persistent_workers=False,\n        generator=generator,\n    )\n\n\npilot_config = {\n    \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n    \'pilot_version\': PILOT_VERSION,\n    \'code_revision\': PILOT_CODE_REVISION,\n    \'fold\': HELDOUT_FOLD,\n    \'fit_subject_count\': len(fold0_fit_df),\n    \'heldout_subject_count\': len(fold0_heldout_df),\n    \'fit_subject_hash\': pilot_subject_hash(fold0_fit_df[\'subject_id\']),\n    \'heldout_subject_hash\': pilot_subject_hash(fold0_heldout_df[\'subject_id\']),\n    \'multimodal_manifest_sha256\': multimodal_manifest_sha256,\n    \'fold_manifest_sha256\': fold_manifest_sha256,\n    \'loader_gate_sha256\': loader_gate_sha256,\n    \'fivefold_gate_sha256\': fivefold_gate_sha256,\n    \'v2_model_gate_sha256\': v2_model_gate_sha256,\n    \'architecture_sha256\': architecture_sha256,\n    \'architecture_contract_sha256\': architecture_contract_sha256,\n    \'graph_schema_sha256\': safe_schema_sha256,\n    \'fold0_graph_scaler_sha256\': fold0_graph_scaler_sha256,\n    \'fold0_clinical_scaler_sha256\': fold0_clinical_scaler_sha256,\n    \'parameters\': EXPECTED_V2_PARAMETERS,\n    \'patch_size\': list(PILOT_PATCH_SIZE),\n    \'samples_per_epoch\': len(training_dataset),\n    \'batch_size\': BATCH_SIZE,\n    \'gradient_accumulation_steps\': ACCUMULATION_STEPS,\n    \'maximum_epochs\': MAX_EPOCHS,\n    \'base_learning_rate\': BASE_LEARNING_RATE,\n    \'gate_learning_rate_multiplier\': GATE_LEARNING_RATE_MULTIPLIER,\n    \'weight_decay\': WEIGHT_DECAY,\n    \'gradient_clip_norm\': GRADIENT_CLIP_NORM,\n    \'ema_decay\': EMA_DECAY,\n    \'early_stopping_minimum_epoch\': EARLY_STOPPING_MINIMUM_EPOCH,\n    \'early_stopping_patience\': PATIENCE_VALIDATIONS,\n    \'early_stopping_minimum_delta\': EARLY_STOPPING_MINIMUM_DELTA,\n    \'checkpoint_metric\': \'heldout_positive_mean_dice\',\n    \'threshold_grid\': PILOT_THRESHOLDS.tolist(),\n    \'mixed_precision\': \'bfloat16_on_cuda_float32_reductions\',\n    \'spatial_augmentation_maximum_attempts\': 4,\n    \'locked_validation_targets_read\': False,\n    \'test_targets_read\': False,\n    \'full_fivefold_training_authorized\': False,\n}\nif PILOT_CONFIG_PATH.exists():\n    existing_config = json.loads(PILOT_CONFIG_PATH.read_text(encoding=\'utf-8\'))\n    immutable_keys = [\n        \'pilot_version\', \'code_revision\', \'fold\',\n        \'fit_subject_hash\', \'heldout_subject_hash\',\n        \'multimodal_manifest_sha256\', \'fold_manifest_sha256\',\n        \'loader_gate_sha256\', \'fivefold_gate_sha256\',\n        \'architecture_contract_sha256\', \'graph_schema_sha256\',\n        \'fold0_graph_scaler_sha256\', \'fold0_clinical_scaler_sha256\',\n        \'parameters\', \'patch_size\', \'samples_per_epoch\', \'batch_size\',\n        \'gradient_accumulation_steps\', \'maximum_epochs\',\n        \'base_learning_rate\', \'gate_learning_rate_multiplier\',\n        \'weight_decay\', \'gradient_clip_norm\', \'ema_decay\',\n        \'early_stopping_minimum_epoch\', \'early_stopping_patience\',\n        \'early_stopping_minimum_delta\', \'checkpoint_metric\',\n        \'threshold_grid\', \'mixed_precision\',\n        \'spatial_augmentation_maximum_attempts\',\n    ]\n    if any(\n        existing_config.get(key) != pilot_config.get(key)\n        for key in immutable_keys\n    ):\n        raise RuntimeError(\'Existing fold-0 pilot configuration changed.\')\nelse:\n    atomic_write_json(pilot_config, PILOT_CONFIG_PATH)\n\nstart_epoch = 1\nglobal_update_step = 0\nbest_mean_dice = -math.inf\nbest_epoch = 0\nbest_threshold = 0.50\nvalidations_without_improvement = 0\nhistory_rows = []\nif PILOT_LAST_CHECKPOINT_PATH.exists():\n    checkpoint = torch.load(\n        PILOT_LAST_CHECKPOINT_PATH,\n        map_location=\'cpu\',\n        weights_only=False,\n    )\n    checkpoint_checks = {\n        \'pilot_version_matches\': checkpoint.get(\'pilot_version\') == PILOT_VERSION,\n        \'code_revision_matches\': checkpoint.get(\'code_revision\')\n        == PILOT_CODE_REVISION,\n        \'manifest_hash_matches\': checkpoint.get(\'multimodal_manifest_sha256\')\n        == multimodal_manifest_sha256,\n        \'loader_gate_hash_matches\': checkpoint.get(\'loader_gate_sha256\')\n        == loader_gate_sha256,\n        \'graph_scaler_hash_matches\': checkpoint.get(\n            \'fold0_graph_scaler_sha256\'\n        ) == fold0_graph_scaler_sha256,\n        \'clinical_scaler_hash_matches\': checkpoint.get(\n            \'fold0_clinical_scaler_sha256\'\n        ) == fold0_clinical_scaler_sha256,\n        \'architecture_contract_hash_matches\': checkpoint.get(\n            \'architecture_contract_sha256\'\n        ) == architecture_contract_sha256,\n        \'parameter_count_matches\': checkpoint.get(\'parameters\')\n        == EXPECTED_V2_PARAMETERS,\n        \'locked_validation_targets_not_read\': checkpoint.get(\n            \'locked_validation_targets_read\'\n        ) is False,\n        \'test_targets_not_read\': checkpoint.get(\'test_targets_read\') is False,\n    }\n    if not all(checkpoint_checks.values()):\n        failed = [name for name, passed in checkpoint_checks.items() if not passed]\n        raise RuntimeError(\'Fold-0 checkpoint mismatch: \' + \', \'.join(failed))\n    training_model.load_state_dict(checkpoint[\'model_state_dict\'])\n    ema_model.load_state_dict(checkpoint[\'ema_model_state_dict\'])\n    training_optimizer.load_state_dict(checkpoint[\'optimizer_state_dict\'])\n    training_scheduler.load_state_dict(checkpoint[\'scheduler_state_dict\'])\n    pilot_restore_rng_state(checkpoint[\'rng_state\'])\n    start_epoch = int(checkpoint[\'epoch\']) + 1\n    global_update_step = int(checkpoint[\'global_update_step\'])\n    best_mean_dice = float(checkpoint[\'best_mean_dice\'])\n    best_epoch = int(checkpoint[\'best_epoch\'])\n    best_threshold = float(checkpoint[\'best_threshold\'])\n    validations_without_improvement = int(\n        checkpoint[\'validations_without_improvement\']\n    )\n    history_rows = list(checkpoint.get(\'history_rows\', []))\n    print(\n        f\'Resuming fold-0 pilot after epoch {start_epoch - 1}; \'\n        f\'best EMA Dice={best_mean_dice:.4f} at epoch {best_epoch}.\'\n    )\nelse:\n    print(\'Starting fold-0 pilot from a fresh deterministic initialization.\')\n\n\n\n\n\n\nprint(\'\\n\' + \'=\' * 80)\nprint(\'TRIAL8 VASCUFATE-NET V2 FOLD-0 PILOT TRAINING START\')\nprint(\'=\' * 80)\nprint(f\'Device / GPU:                       {device} / \' + (\n    torch.cuda.get_device_name(0) if device.type == \'cuda\' else \'CPU\'\n))\nprint(f\'Fit / held-out / locked val / test: {len(fold0_fit_df)} / {len(fold0_heldout_df)} / 0 / 0 targets\')\nprint(f\'Patch size:                         {PILOT_PATCH_SIZE}\')\nprint(f\'Samples per epoch:                  {len(training_dataset)} (112/28/28)\')\nprint(f\'Batch / accumulation / effective:   {BATCH_SIZE} / {ACCUMULATION_STEPS} / {BATCH_SIZE * ACCUMULATION_STEPS}\')\nprint(f\'Maximum epochs:                     {MAX_EPOCHS}\')\nprint(f\'Early stopping:                     epoch >= {EARLY_STOPPING_MINIMUM_EPOCH}, patience={PATIENCE_VALIDATIONS}, delta={EARLY_STOPPING_MINIMUM_DELTA:.3f}\')\nprint(f\'Base/gate learning rates:           {BASE_LEARNING_RATE:.1e} / {BASE_LEARNING_RATE * GATE_LEARNING_RATE_MULTIPLIER:.1e}\')\nprint(f\'Total parameters:                   {EXPECTED_V2_PARAMETERS:,}\')\nprint(\'Validation:                         full-volume EMA every epoch\')\nprint(\'Threshold calibration:              held-out fold only, 0.10..0.80\')\nprint(\'Locked validation/test targets:     sealed\')\n\ntraining_already_complete = (\n    start_epoch > MAX_EPOCHS\n    or (\n        start_epoch - 1 >= EARLY_STOPPING_MINIMUM_EPOCH\n        and validations_without_improvement >= PATIENCE_VALIDATIONS\n    )\n)\nstop_reason = (\n    \'already_complete_on_resume\'\n    if training_already_complete\n    else \'maximum_epochs_reached\'\n)\n\nfor epoch in range(\n    start_epoch,\n    start_epoch if training_already_complete else MAX_EPOCHS + 1,\n):\n    epoch_start = datetime.now(timezone.utc)\n    loader = pilot_epoch_loader(epoch)\n    expected_counts = dict(training_dataset.counts)\n    training_model.train()\n    training_optimizer.zero_grad(set_to_none=True)\n    running_names = [\n        \'total_loss\', \'main_loss\', \'soft_dice_loss\',\n        \'soft_focal_bce_loss\', \'hard_focal_tversky_loss\',\n        \'hard_contribution\', \'half_loss\', \'quarter_loss\', \'node_loss\',\n        \'atlas_retained_fraction\', \'vascular_retained_fraction\',\n        \'occlusion_retained_fraction\', \'clinical_retained_fraction\',\n        \'spatially_augmented_fraction\',\n    ]\n    running = {name: 0.0 for name in running_names}\n    processed_batches = 0\n    processed_samples = 0\n    observed_counts = {\n        \'positive\': 0, \'topology_negative\': 0, \'brain_negative\': 0,\n    }\n    gradient_norms = []\n    accumulation_subjects = []\n    for batch_index, cpu_batch in enumerate(loader, start=1):\n        processed_samples += int(cpu_batch[\'target_binary\'].shape[0])\n        for sample_type in cpu_batch[\'sample_type\']:\n            observed_counts[str(sample_type)] += 1\n        accumulation_subjects.extend(str(value) for value in cpu_batch[\'subject_id\'])\n        with torch.autocast(\n            device_type=device.type,\n            dtype=torch.bfloat16,\n            enabled=device.type == \'cuda\',\n        ):\n            total_loss, parts = pilot_compute_training_loss(\n                training_model, cpu_batch\n            )\n            group_start = ((batch_index - 1) // ACCUMULATION_STEPS) * (\n                ACCUMULATION_STEPS\n            )\n            group_size = min(\n                ACCUMULATION_STEPS, len(loader) - group_start\n            )\n            scaled_loss = total_loss / group_size\n        scaled_loss.backward()\n        for name in running_names:\n            running[name] += float(parts[name].detach().cpu())\n        processed_batches += 1\n        should_update = (\n            batch_index % ACCUMULATION_STEPS == 0\n            or batch_index == len(loader)\n        )\n        if should_update:\n            gradient_norm = float(torch.nn.utils.clip_grad_norm_(\n                training_model.parameters(), GRADIENT_CLIP_NORM\n            ).item())\n            if not math.isfinite(gradient_norm):\n                training_optimizer.zero_grad(set_to_none=True)\n                raise RuntimeError(\n                    f\'Non-finite gradient at epoch {epoch}, batch \'\n                    f\'{batch_index}, subjects={accumulation_subjects}.\'\n                )\n            gradient_norms.append(gradient_norm)\n            training_optimizer.step()\n            training_optimizer.zero_grad(set_to_none=True)\n            training_scheduler.step()\n            global_update_step += 1\n            pilot_update_ema(\n                ema_model, training_model, EMA_DECAY, global_update_step\n            )\n            accumulation_subjects.clear()\n    if observed_counts != expected_counts or processed_samples != len(training_dataset):\n        raise RuntimeError(\n            f\'Epoch {epoch}: sampling counts changed. \'\n            f\'Observed {observed_counts}; expected {expected_counts}.\'\n        )\n    epoch_means = {\n        name: running[name] / max(processed_batches, 1)\n        for name in running_names\n    }\n    if not gradient_norms or not all(\n        math.isfinite(value) for value in epoch_means.values()\n    ):\n        raise RuntimeError(f\'Epoch {epoch}: non-finite training state.\')\n\n    print(f\'Running full fold-0 EMA validation at epoch {epoch}...\')\n    validation_result = pilot_validate(ema_model, epoch)\n    current_dice = float(validation_result[\'positive_mean_dice\'])\n    current_threshold = float(validation_result[\'threshold\'])\n    improved = current_dice > best_mean_dice + EARLY_STOPPING_MINIMUM_DELTA\n    if improved:\n        best_mean_dice = current_dice\n        best_epoch = epoch\n        best_threshold = current_threshold\n        validations_without_improvement = 0\n    else:\n        validations_without_improvement += 1\n    epoch_seconds = (\n        datetime.now(timezone.utc) - epoch_start\n    ).total_seconds()\n    history_row = {\n        \'epoch\': epoch,\n        \'global_update_step\': global_update_step,\n        \'standard_learning_rate\': float(training_optimizer.param_groups[0][\'lr\']),\n        \'fusion_gate_learning_rate\': float(training_optimizer.param_groups[1][\'lr\']),\n        **epoch_means,\n        \'mean_gradient_norm\': float(np.mean(gradient_norms)),\n        \'maximum_gradient_norm\': float(np.max(gradient_norms)),\n        \'positive_patch_fraction\': observed_counts[\'positive\'] / processed_samples,\n        \'topology_negative_fraction\': (\n            observed_counts[\'topology_negative\'] / processed_samples\n        ),\n        \'brain_negative_fraction\': (\n            observed_counts[\'brain_negative\'] / processed_samples\n        ),\n        \'validation_threshold\': current_threshold,\n        \'validation_positive_mean_dice\': current_dice,\n        \'validation_positive_median_dice\': validation_result[\n            \'positive_median_dice\'\n        ],\n        \'validation_positive_global_dice\': validation_result[\n            \'positive_global_dice\'\n        ],\n        \'validation_mean_voxel_precision\': validation_result[\n            \'mean_voxel_precision\'\n        ],\n        \'validation_mean_voxel_recall\': validation_result[\n            \'mean_voxel_recall\'\n        ],\n        \'validation_lesion_precision\': validation_result[\'lesion_precision\'],\n        \'validation_lesion_recall\': validation_result[\'lesion_recall\'],\n        \'validation_lesion_f1\': validation_result[\'lesion_f1\'],\n        \'validation_mean_absolute_volume_error_ml\': validation_result[\n            \'mean_absolute_volume_error_ml\'\n        ],\n        \'validation_mean_empty_false_positive_volume_ml\': validation_result[\n            \'mean_empty_false_positive_volume_ml\'\n        ],\n        \'best_positive_mean_dice\': best_mean_dice,\n        \'best_epoch\': best_epoch,\n        \'best_threshold\': best_threshold,\n        \'validations_without_improvement\': validations_without_improvement,\n        \'improved\': improved,\n        \'epoch_seconds\': epoch_seconds,\n        \'locked_validation_targets_read\': False,\n        \'test_targets_read\': False,\n    }\n    for lesion_bin, value in validation_result[\'lesion_bin_mean_dice\'].items():\n        history_row[f\'validation_mean_dice_{lesion_bin}\'] = value\n    history_rows.append(to_builtin(history_row))\n    atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)\n\n    checkpoint_record = {\n        \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'pilot_version\': PILOT_VERSION,\n        \'code_revision\': PILOT_CODE_REVISION,\n        \'epoch\': epoch,\n        \'global_update_step\': global_update_step,\n        \'model_state_dict\': training_model.state_dict(),\n        \'ema_model_state_dict\': ema_model.state_dict(),\n        \'optimizer_state_dict\': training_optimizer.state_dict(),\n        \'scheduler_state_dict\': training_scheduler.state_dict(),\n        \'rng_state\': pilot_capture_rng_state(),\n        \'best_mean_dice\': best_mean_dice,\n        \'best_epoch\': best_epoch,\n        \'best_threshold\': best_threshold,\n        \'validations_without_improvement\': validations_without_improvement,\n        \'history_rows\': history_rows,\n        \'multimodal_manifest_sha256\': multimodal_manifest_sha256,\n        \'loader_gate_sha256\': loader_gate_sha256,\n        \'fivefold_gate_sha256\': fivefold_gate_sha256,\n        \'architecture_sha256\': architecture_sha256,\n        \'architecture_contract_sha256\': architecture_contract_sha256,\n        \'fold0_graph_scaler_sha256\': fold0_graph_scaler_sha256,\n        \'fold0_clinical_scaler_sha256\': fold0_clinical_scaler_sha256,\n        \'parameters\': EXPECTED_V2_PARAMETERS,\n        \'locked_validation_targets_read\': False,\n        \'test_targets_read\': False,\n        \'outcome_workbooks_read\': False,\n    }\n    pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)\n    if improved:\n        pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)\n\n    print(\n        f\'Epoch {epoch:03d}/{MAX_EPOCHS}: \'\n        f\'loss={epoch_means["total_loss"]:.4f}, \'\n        f\'main={epoch_means["main_loss"]:.4f}, \'\n        f\'node={epoch_means["node_loss"]:.4f}, \'\n        f\'lr={history_row["standard_learning_rate"]:.2e}, \'\n        f\'gate-lr={history_row["fusion_gate_learning_rate"]:.2e}, \'\n        f\'grad={history_row["mean_gradient_norm"]:.3f}, \'\n        f\'time={epoch_seconds / 60.0:.1f} min\'\n    )\n    print(\n        f\'  EMA held-out: mean Dice={current_dice:.4f}, \'\n        f\'median Dice={validation_result["positive_median_dice"]:.4f}, \'\n        f\'global Dice={validation_result["positive_global_dice"]:.4f}, \'\n        f\'voxel P/R={validation_result["mean_voxel_precision"]:.4f}/\'\n        f\'{validation_result["mean_voxel_recall"]:.4f}, \'\n        f\'lesion F1={validation_result["lesion_f1"]:.4f}, \'\n        f\'threshold={current_threshold:.2f}\'\n    )\n    print(\n        f\'  Volume MAE={validation_result["mean_absolute_volume_error_ml"]:.3f} mL, \'\n        f\'empty FP={validation_result["mean_empty_false_positive_volume_ml"]:.3f} mL, \'\n        f\'best={best_mean_dice:.4f} at epoch {best_epoch}, \'\n        f\'patience={validations_without_improvement}/{PATIENCE_VALIDATIONS}\'\n    )\n    printable_bins = [\n        f\'{name}={value:.4f}\'\n        for name, value in validation_result[\'lesion_bin_mean_dice\'].items()\n        if math.isfinite(value)\n    ]\n    print(\'  Mean Dice by lesion bin: \' + \', \'.join(printable_bins))\n\n    atomic_write_json({\n        \'updated_at_utc\': datetime.now(timezone.utc).isoformat(),\n        \'pilot_version\': PILOT_VERSION,\n        \'status\': \'running\',\n        \'last_completed_epoch\': epoch,\n        \'best_positive_mean_dice\': best_mean_dice,\n        \'best_epoch\': best_epoch,\n        \'best_threshold\': best_threshold,\n        \'validations_without_improvement\': validations_without_improvement,\n        \'locked_validation_targets_read\': False,\n        \'test_targets_read\': False,\n        \'full_fivefold_training_authorized\': False,\n        \'last_checkpoint_path\': str(PILOT_LAST_CHECKPOINT_PATH),\n        \'best_checkpoint_path\': str(PILOT_BEST_CHECKPOINT_PATH),\n    }, PILOT_STATE_PATH)\n    if (\n        epoch >= EARLY_STOPPING_MINIMUM_EPOCH\n        and validations_without_improvement >= PATIENCE_VALIDATIONS\n    ):\n        stop_reason = \'early_stopping\'\n        print(\n            f\'Early stopping after {validations_without_improvement} \'\n            \'held-out validations without sufficient improvement.\'\n        )\n        break\n\ncompleted_epoch = int(history_rows[-1][\'epoch\']) if history_rows else start_epoch - 1\npilot_complete = bool(\n    completed_epoch >= MAX_EPOCHS or stop_reason in {\n        \'early_stopping\', \'already_complete_on_resume\'\n    }\n)\nresult_gate = {\n    \'updated_at_utc\': datetime.now(timezone.utc).isoformat(),\n    \'pilot_version\': PILOT_VERSION,\n    \'code_revision\': PILOT_CODE_REVISION,\n    \'pilot_complete\': pilot_complete,\n    \'stop_reason\': stop_reason,\n    \'last_completed_epoch\': completed_epoch,\n    \'best_positive_mean_dice\': best_mean_dice,\n    \'best_epoch\': best_epoch,\n    \'best_threshold\': best_threshold,\n    \'fit_subjects\': len(fold0_fit_df),\n    \'heldout_subjects\': len(fold0_heldout_df),\n    \'parameters\': EXPECTED_V2_PARAMETERS,\n    \'locked_validation_targets_read\': False,\n    \'test_targets_read\': False,\n    \'outcome_workbooks_read\': False,\n    \'pilot_review_required\': True,\n    \'full_fivefold_training_authorized\': False,\n    \'best_checkpoint_path\': str(PILOT_BEST_CHECKPOINT_PATH),\n    \'last_checkpoint_path\': str(PILOT_LAST_CHECKPOINT_PATH),\n    \'history_path\': str(PILOT_HISTORY_PATH),\n}\natomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)\natomic_write_json({\n    **result_gate,\n    \'status\': \'complete\' if pilot_complete else \'interrupted\',\n}, PILOT_STATE_PATH)\n\nprint(\'\\n\' + \'=\' * 80)\nprint(\'TRIAL8 VASCUFATE-NET V2 FOLD-0 PILOT COMPLETE\')\nprint(\'=\' * 80)\nprint(f\'Stop reason:                        {stop_reason}\')\nprint(f\'Last completed epoch:               {completed_epoch}\')\nprint(f\'Best EMA positive mean Dice:        {best_mean_dice:.4f}\')\nprint(f\'Best epoch:                         {best_epoch}\')\nprint(f\'Calibrated threshold:               {best_threshold:.2f}\')\nprint(\'Validation cadence:                 every epoch\')\nprint(\'Locked validation targets read:     False\')\nprint(\'Test targets read:                  False\')\nprint(\'Full five-fold training authorized: False (pilot review required)\')\nprint(f\'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}\')\nprint(f\'Last checkpoint:                    {PILOT_LAST_CHECKPOINT_PATH}\')\nprint(f\'History:                            {PILOT_HISTORY_PATH}\')\nprint(f\'Result gate:                        {PILOT_RESULT_GATE_PATH}\')\n'
del _INERT_ORIGINAL_PILOT_TRAINING_PROVENANCE


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            '-q',
            pip_name or import_name,
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
ensure_package('scipy')

from scipy import ndimage
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
PREFLIGHT_VERSION = 'v1_component_proposal_arterial_projection_contract'
CODE_REVISION = 'r3_sampling_mask_diagnostic_not_target_boundary'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)
EXPECTED_DIAGNOSTIC_CODE_REVISION = (
    'r1_branch_component_threshold_overfit_audit'
)

FOLD0 = 0
SEED = 20260930
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_DIVISOR = (32, 32, 32)




GLOBAL_CANVAS_LIMIT = (288, 320, 128)
GLOBAL_IMAGE_DOWNSAMPLE = (2, 2, 2)
GLOBAL_PROPOSAL_FACTOR = (2, 2, 2)
LOCAL_PROPOSAL_FACTOR = (2, 2, 1)
PROPOSAL_CLUSTER_DILATION_MM = 1.6
POSITIVE_PATCHES_PER_EPOCH = 112
TOPOLOGY_NEGATIVE_PATCHES_PER_EPOCH = 28
BRAIN_NEGATIVE_PATCHES_PER_EPOCH = 28
MAX_POSITIVE_PATCHES_PER_SUBJECT_PER_EPOCH = 2
AUDIT_EPOCHS = (0, 1)

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT
    / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free'
    / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
FIVEFOLD_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
    / 'fivefold_loader_loss_sampler_gate.json'
)
V2_ARCHITECTURE_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
DIAGNOSTIC_GATE_PATH = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'model_diagnostics'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v2_2_fold0_failure_diagnostic_r1'
    / 'diagnostic_gate.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / 'preprocessing'
    / R2_VERSION
    / 'architecture_preflight'
    / MODEL_VERSION
    / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FIT_SUBJECT_AUDIT_PATH = OUTPUT_ROOT / 'fold0_fit_subject_audit.csv'
COMPONENT_AUDIT_PATH = OUTPUT_ROOT / 'fold0_fit_component_audit.csv'
COMPONENT_MASK_DISAGREEMENT_PATH = (
    OUTPUT_ROOT / 'fold0_component_sampling_mask_disagreement.csv'
)
COMPONENT_INDEX_PATH = OUTPUT_ROOT / 'fold0_fit_component_index.npz'
SCHEDULE_PATH = OUTPUT_ROOT / 'fold0_component_sampling_schedule.csv'
SCHEDULE_SUMMARY_PATH = OUTPUT_ROOT / 'fold0_sampling_schedule_summary.csv'
GLOBAL_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'global_view_shape_audit.csv'
ARTERIAL_AUDIT_PATH = OUTPUT_ROOT / 'arterial_projection_audit.csv'
ARTERIAL_TOPOLOGY_PATH = OUTPUT_ROOT / 'arterial_projection_topology.csv'
ARCHITECTURE_CONTRACT_PATH = OUTPUT_ROOT / 'v2_2_architecture_contract.json'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_block1_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_block1_preflight_gate.json'

FORBIDDEN_SIDECAR_KEYS = {
    'target_binary',
    'target_soft',
    'binary_loss_weight',
    'source_target_volume_ml',
    'target_binary_volume_ml',
    'target_soft_volume_ml',
    'target_crop_coverage',
}

EXPECTED_CROSS_RELATIONS = [
    'primary_supply',
    'communicating_or_proximal_supply',
]
ARTERIAL_SOURCE_FEATURES = [
    'cta_mean',
    'cta_p90',
    'cbf_mean',
    'tmax_mean',
    'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
ARTERIAL_RELATION_PRIOR = {
    'primary_supply': 1.0,
    'communicating_or_proximal_supply': 0.5,
}

ARCHITECTURE_CONTRACT = {
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'problem_statement': (
        'V2/V2.1 are representation- and localization-limited: most missed '
        'components never receive meaningful probability, while threshold '
        'or hysteresis changes do not recover them.'
    ),
    'reference_path': {
        'preserved_architecture': 'VascuFateNet-v2 local refinement path',
        'reference_parameters': EXPECTED_V2_PARAMETERS,
        'checkpoint_warm_start': False,
        'reason': (
            'preserve the tested local backbone design without inheriting '
            'wrong-site V2/V2.1 pilot activations'
        ),
    },
    'global_to_local_design': {
        'global_input': (
            'symmetric zero-pad to frozen full-brain canvas, then '
            'anti-aliased 2x2x2 downsample'
        ),
        'global_canvas_divisor': list(GLOBAL_CANVAS_DIVISOR),
        'global_canvas_limit': list(GLOBAL_CANVAS_LIMIT),
        'global_image_downsample': list(GLOBAL_IMAGE_DOWNSAMPLE),
        'global_outputs': [
            'coarse lesion occupancy logits',
            'component proposal scores',
            'lesion-presence logit',
            'log-volume burden estimate',
        ],
        'local_input_patch': list(LOCAL_PATCH_SIZE),
        'local_role': (
            'high-resolution proposal-conditioned boundary and extent '
            'refinement using the preserved V2 image/vascular backbone'
        ),
        'proposal_to_local_connection': (
            'detached or scheduled-detach proposal centering during early '
            'training; ground-truth jitter remains available for coverage'
        ),
        'direct_global_or_graph_to_final_logit_residual': False,
    },
    'component_supervision': {
        'atomic_definition': '26-connected components on sampling support',
        'hard_and_soft_support': 'target_binary > 0',
        'soft_only_subvoxel_support': 'target_soft > 0',
        'true_empty_support': 'empty',
        'proposal_cluster_definition': (
            'atomic components joined only after 1.6-mm physical dilation; '
            'atomic components remain immutable evaluation units'
        ),
        'global_proposal_factor': list(GLOBAL_PROPOSAL_FACTOR),
        'local_proposal_factor': list(LOCAL_PROPOSAL_FACTOR),
        'downsample_rule': 'max occupancy, never nearest or mean thresholding',
        'crop_mask_semantics': (
            'target-independent brain/negative-sampling mask; disagreement '
            'with a lesion component is audited but cannot erase or invalidate '
            'authoritative target supervision'
        ),
        'geometry_safety_rule': (
            'targets must share the final image grid, remain within the array, '
            'avoid array-edge truncation, and survive both proposal scales'
        ),
        'component_mil_rule': (
            'every training component contributes an independent positive '
            'bag; larger components cannot absorb the supervision of smaller '
            'components in the same subject'
        ),
    },
    'sampling_policy': {
        'fit_partition': 'fold-0 fit only: train subjects with fold != 0',
        'positive_patches_per_epoch': POSITIVE_PATCHES_PER_EPOCH,
        'topology_negative_patches_per_epoch': (
            TOPOLOGY_NEGATIVE_PATCHES_PER_EPOCH
        ),
        'brain_negative_patches_per_epoch': BRAIN_NEGATIVE_PATCHES_PER_EPOCH,
        'positive_fraction': 2.0 / 3.0,
        'all_positive_subjects_each_epoch': True,
        'maximum_positive_patches_per_subject_per_epoch': (
            MAX_POSITIVE_PATCHES_PER_SUBJECT_PER_EPOCH
        ),
        'extra_positive_allocation': (
            'round-robin across subject-level lesion-volume bins, then '
            'inverse-component-volume selection within subject'
        ),
        'soft_only_component_alternation': True,
        'volume_based_case_loss_weights': False,
        'top_k_ohem': False,
    },
    'directed_arterial_projection': {
        'edge_direction': 'vessel node -> vascular territory node',
        'relations': EXPECTED_CROSS_RELATIONS,
        'relation_prior': ARTERIAL_RELATION_PRIOR,
        'source_features': ARTERIAL_SOURCE_FEATURES,
        'source_validity': (
            'vessel presence AND feature validity; target territory validity '
            'AND reliability; atlas, registration, and perfusion confidence'
        ),
        'spatial_projection': (
            'territory risk is broadcast only through the registered '
            'intracranially clipped territory map'
        ),
        'atlas_disabled_behavior': 'risk and projection-validity are zero',
        'fusion': 'bounded, confidence-gated, zero-initialized feature fusion',
        'direct_graph_to_lesion_logit': False,
        'venous_topology_claimed': False,
    },
    'clinical_role': {
        'input_dimension': 44,
        'allowed_outputs': [
            'lesion-presence prior',
            'log-volume burden prior',
            'bounded global proposal temperature/bias',
        ],
        'spatial_localization_allowed': False,
        'direct_lesion_logit_allowed': False,
        'post_imaging_treatment_fields_allowed': False,
        'post_treatment_outcomes_allowed': False,
        'treatment_conditioned_extension': (
            'separate secondary study only, with a deployment-time treatment '
            'contract and no reperfusion-outcome leakage'
        ),
    },
    'planned_loss_allocation': {
        'local_hard_soft_segmentation': 0.55,
        'component_mil_and_proposal_recall': 0.20,
        'global_occupancy_focal': 0.15,
        'local_boundary_refinement': 0.05,
        'presence_and_log_volume_burden': 0.05,
    },
    'training_stages_for_next_block': [
        'stage A: global proposal/presence/burden heads with local path frozen',
        'stage B: joint proposal-conditioned local refinement',
        'stage C: low-learning-rate end-to-end calibration with EMA',
    ],
    'parameter_budget': {
        'reference_v2_parameters': EXPECTED_V2_PARAMETERS,
        'maximum_new_parameters': 2_500_000,
        'maximum_total_parameters': EXPECTED_V2_PARAMETERS + 2_500_000,
    },
    'diagnostic_outputs_required_each_epoch': [
        'positive mean, median, and global Dice',
        'Dice by lesion-volume bin',
        'official one-to-one IoU20 lesion precision, recall, and F1',
        'component proposal recall at 0.03, 0.10, 0.30, and 0.50',
        'representation-failure component fraction',
        'component center-distance distribution',
        'merge, split, and fragmentation counts',
        'proposal-stage versus final-stage component recall',
        'absolute lesion-volume error',
        'branch gate magnitudes and controlled ablations',
    ],
    'predeclared_fold0_mechanism_gate': {
        'minimum_positive_mean_dice': 0.258,
        'minimum_small_lesion_mean_dice': 0.080,
        'minimum_official_lesion_recall_iou20': 0.065,
        'minimum_official_lesion_f1_iou20': 0.080,
        'maximum_representation_failure_fraction': 0.50,
        'maximum_large_or_very_large_mean_dice_drop': 0.020,
        'interpretation': (
            'architecture-mechanism gate only; it is not a SOTA claim'
        ),
    },
    'later_competitive_gate': {
        'target_mean_dice': 0.30,
        'target_official_lesion_f1_iou20': 0.15,
        'requires_fivefold_and_locked_validation_confirmation': True,
        'requires_matched_strong_image_baseline': True,
    },
    'leakage_policy': {
        'block1_target_access': 'fold-0 fitting targets only',
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
    },
    'model_constructed_in_this_block': False,
    'training_performed_in_this_block': False,
    'model_training_authorized': False,
}






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return True
    if text in {'false', '0', 'no', '', 'nan', 'none'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def decode_strings(array):
    values = np.asarray(array).reshape(-1)
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in values
    ]


def scalar(data, name, default=np.nan):
    if name not in data.files:
        return float(default)
    return float(np.asarray(data[name]).reshape(-1)[0])


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def canonical_json_sha256(record):
    payload = json.dumps(
        to_builtin(record), sort_keys=True, separators=(',', ':')
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def atomic_write_json(record, path):
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(to_builtin(record), indent=2, sort_keys=True),
        encoding='utf-8',
    )
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def atomic_write_npz(path, **arrays):
    temporary = Path(str(path) + '.tmp')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def npz_array_header(path, key):
    member_name = f'{key}.npy'
    with zipfile.ZipFile(path, 'r') as archive:
        if member_name not in archive.namelist():
            raise KeyError(f'{path}: missing NPZ member {member_name}')
        with archive.open(member_name, 'r') as handle:
            version = np.lib.format.read_magic(handle)
            if version == (1, 0):
                shape, fortran, dtype = np.lib.format.read_array_header_1_0(
                    handle
                )
            elif version == (2, 0):
                shape, fortran, dtype = np.lib.format.read_array_header_2_0(
                    handle
                )
            elif version == (3, 0):
                shape, fortran, dtype = np.lib.format._read_array_header(
                    handle, version
                )
            else:
                raise RuntimeError(
                    f'{path}: unsupported NPY member version {version}'
                )
    return tuple(int(value) for value in shape), bool(fortran), np.dtype(dtype)


def round_up(value, divisor):
    return int(math.ceil(int(value) / int(divisor)) * int(divisor))


def subject_hash(subjects):
    payload = '\n'.join(sorted(str(value) for value in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


CONTRACT_SHA256 = canonical_json_sha256(ARCHITECTURE_CONTRACT)






required_paths = [
    TARGET_POLICY_MANIFEST_PATH,
    FOLD_MANIFEST_PATH,
    MULTIMODAL_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH,
    V2_ARCHITECTURE_GATE_PATH,
    MULTIMODAL_GATE_PATH,
    DIAGNOSTIC_GATE_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 Block 1 prerequisites are missing:\n'
        + '\n'.join(missing_paths)
    )

observed_hashes = {
    'target_policy_manifest': file_sha256(TARGET_POLICY_MANIFEST_PATH),
    'fold_manifest': file_sha256(FOLD_MANIFEST_PATH),
    'multimodal_manifest': file_sha256(MULTIMODAL_MANIFEST_PATH),
    'graph_schema': file_sha256(GRAPH_SCHEMA_PATH),
}
expected_hashes = {
    'target_policy_manifest': EXPECTED_TARGET_POLICY_MANIFEST_SHA256,
    'fold_manifest': EXPECTED_FOLD_MANIFEST_SHA256,
    'multimodal_manifest': EXPECTED_MULTIMODAL_MANIFEST_SHA256,
    'graph_schema': EXPECTED_GRAPH_SCHEMA_SHA256,
}
hash_checks = {
    f'{name}_hash_matches': observed_hashes[name] == expected
    for name, expected in expected_hashes.items()
}
if not all(hash_checks.values()):
    failed = [name for name, passed in hash_checks.items() if not passed]
    raise RuntimeError('Frozen input hash mismatch: ' + ', '.join(failed))

fivefold_gate = json.loads(FIVEFOLD_GATE_PATH.read_text(encoding='utf-8'))
v2_architecture_gate = json.loads(
    V2_ARCHITECTURE_GATE_PATH.read_text(encoding='utf-8')
)
multimodal_gate = json.loads(
    MULTIMODAL_GATE_PATH.read_text(encoding='utf-8')
)
diagnostic_gate = json.loads(
    DIAGNOSTIC_GATE_PATH.read_text(encoding='utf-8')
)

upstream_checks = {
    **hash_checks,
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'fivefold_contract_verified': (
        fivefold_gate.get('baseline_loader_loss_sampler_contract_verified')
        is True
    ),
    'fivefold_training_remained_locked': (
        fivefold_gate.get('fold0_pilot_training_authorized') is False
        and fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'v2_model_preflight_passed': (
        v2_architecture_gate.get('preflight_passed') is True
    ),
    'v2_parameter_count_matches': int(
        v2_architecture_gate.get('total_parameters', -1)
    ) == EXPECTED_V2_PARAMETERS,
    'v2_training_remained_locked': (
        v2_architecture_gate.get('model_training_authorized') is False
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'multimodal_manifest_hash_matches_gate': (
        multimodal_gate.get('candidate_manifest_sha256')
        == observed_hashes['multimodal_manifest']
    ),
    'multimodal_pilot_implementation_allowed': (
        multimodal_gate.get('pilot_training_implementation_authorized') is True
    ),
    'multimodal_training_remained_locked': (
        multimodal_gate.get('fold0_pilot_training_authorized') is False
        and multimodal_gate.get('full_fivefold_training_authorized') is False
    ),
    'failure_diagnostic_passed': (
        diagnostic_gate.get('diagnostic_passed') is True
    ),
    'failure_diagnostic_revision_matches': (
        diagnostic_gate.get('code_revision')
        == EXPECTED_DIAGNOSTIC_CODE_REVISION
    ),
    'diagnostic_performed_no_training': (
        diagnostic_gate.get('training_performed') is False
        and int(diagnostic_gate.get('study_trainable_parameters', -1)) == 0
    ),
    'diagnostic_kept_locked_targets_sealed': (
        diagnostic_gate.get('locked_validation_targets_read') is False
        and diagnostic_gate.get('test_targets_read') is False
    ),
    'diagnostic_read_no_outcomes': (
        diagnostic_gate.get('outcome_workbooks_read') is False
    ),
}
if not all(upstream_checks.values()):
    failed = [name for name, passed in upstream_checks.items() if not passed]
    raise RuntimeError('Trial8 V2.2 upstream gate failure: ' + ', '.join(failed))

if PREFLIGHT_GATE_PATH.exists():
    existing_gate = json.loads(PREFLIGHT_GATE_PATH.read_text(encoding='utf-8'))
    expected_outputs = existing_gate.get('output_sha256', {})
    output_hashes_stable = bool(expected_outputs) and all(
        Path(path).exists() and file_sha256(path) == digest
        for path, digest in expected_outputs.items()
    )
    if (
        existing_gate.get('preflight_passed') is True
        and existing_gate.get('code_revision') == CODE_REVISION
        and existing_gate.get('architecture_contract_sha256')
        == CONTRACT_SHA256
        and existing_gate.get('input_sha256') == observed_hashes
        and output_hashes_stable
    ):
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 BLOCK 1 PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
        print('No target, architecture, or output artifact was rewritten.')
        raise SystemExit(0)

print('Trial8 V2.2 Block 1 upstream gates passed.')
print(f'Target-policy manifest SHA-256: {observed_hashes["target_policy_manifest"]}')
print(f'Five-fold manifest SHA-256:     {observed_hashes["fold_manifest"]}')
print(f'Multimodal manifest SHA-256:    {observed_hashes["multimodal_manifest"]}')
print(f'Graph schema SHA-256:           {observed_hashes["graph_schema"]}')
print('Locked validation targets read: False')
print('Test targets read:              False')
print('Study trainable parameters:     0 (architecture preflight only)')






manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
policy_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
fold_df = pd.read_csv(FOLD_MANIFEST_PATH)

required_manifest_columns = {
    'subject_id',
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_atlas_enabled',
    'trial8_atlas_vascular30_path',
    'trial8_graph_sidecar_path',
    'trial8_graph_sidecar_sha256',
    'trial8_supervision_mode',
    'trial8_effective_target_volume_ml',
    'trial8_effective_lesion_volume_bin',
    'trial8_sampling_positive',
    'trial8_hard_metric_eligible',
    'trial8_target_policy_frozen',
    'trial8_v2_multimodal_loader_preflight_passed',
    'trial8_model_training_authorized',
}
required_fold_columns = {
    'subject_id',
    'frozen_dev_split',
    'development_fold',
    'target_policy_manifest_sha256',
}
missing_manifest_columns = sorted(
    required_manifest_columns - set(manifest_df.columns)
)
missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
if missing_manifest_columns or missing_fold_columns:
    raise RuntimeError(
        'Required V2.2 manifest columns are missing. Manifest: '
        f'{missing_manifest_columns}; folds: {missing_fold_columns}'
    )

manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
policy_df = policy_df.sort_values('subject_id').reset_index(drop=True)
fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
subject_order = manifest_df['subject_id'].astype(str).tolist()
if subject_order != policy_df['subject_id'].astype(str).tolist():
    raise RuntimeError('Multimodal and target-policy subject order differs.')
if subject_order != fold_df['subject_id'].astype(str).tolist():
    raise RuntimeError('Multimodal and fold-manifest subject order differs.')

policy_string_columns = [
    'dev_split',
    'vascufate_v2_npz_path',
    'trial8_npz_sha256',
    'trial8_supervision_mode',
    'trial8_effective_lesion_volume_bin',
]
policy_boolean_columns = [
    'trial8_atlas_enabled',
    'trial8_sampling_positive',
    'trial8_hard_metric_eligible',
    'trial8_target_policy_frozen',
]
policy_cross_checks = {}
for column in policy_string_columns:
    policy_cross_checks[f'{column}_matches_target_policy'] = (
        column in policy_df.columns
        and manifest_df[column].astype(str).equals(policy_df[column].astype(str))
    )
for column in policy_boolean_columns:
    policy_cross_checks[f'{column}_matches_target_policy'] = (
        column in policy_df.columns
        and manifest_df[column].map(as_bool).equals(policy_df[column].map(as_bool))
    )
policy_cross_checks['effective_target_volume_matches_target_policy'] = (
    'trial8_effective_target_volume_ml' in policy_df.columns
    and np.allclose(
        manifest_df['trial8_effective_target_volume_ml'].to_numpy(
            dtype=np.float64
        ),
        policy_df['trial8_effective_target_volume_ml'].to_numpy(
            dtype=np.float64
        ),
        rtol=0.0,
        atol=1e-12,
        equal_nan=True,
    )
)
if not all(policy_cross_checks.values()):
    failed = [
        name for name, passed in policy_cross_checks.items() if not passed
    ]
    raise RuntimeError(
        'Multimodal manifest differs from the frozen target policy: '
        + ', '.join(failed)
    )

manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
fold0_fit_df = manifest_df[
    manifest_df['dev_split'].astype(str).eq('train')
    & manifest_df['development_fold'].ne(FOLD0)
].copy()
fold0_heldout_df = manifest_df[
    manifest_df['dev_split'].astype(str).eq('train')
    & manifest_df['development_fold'].eq(FOLD0)
].copy()
locked_df = manifest_df[
    manifest_df['dev_split'].astype(str).isin(['val', 'test'])
].copy()

manifest_checks = {
    **policy_cross_checks,
    'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
    'manifest_subjects_unique': (
        manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
    ),
    'split_counts_match': (
        manifest_df['dev_split'].value_counts().to_dict() == EXPECTED_SPLITS
    ),
    'fold0_fit_has_82_subjects': len(fold0_fit_df) == EXPECTED_FOLD0_FIT,
    'fold0_heldout_has_20_subjects': (
        len(fold0_heldout_df) == EXPECTED_FOLD0_HELDOUT
    ),
    'fold0_fit_and_heldout_are_disjoint': not bool(
        set(fold0_fit_df['subject_id']) & set(fold0_heldout_df['subject_id'])
    ),
    'fold_splits_match_manifest': fold_df['frozen_dev_split'].astype(
        str
    ).equals(manifest_df['dev_split'].astype(str)),
    'fold_target_policy_hash_matches': bool(
        fold_df['target_policy_manifest_sha256'].astype(str).eq(
            observed_hashes['target_policy_manifest']
        ).all()
    ),
    'target_policy_frozen_for_all': bool(
        manifest_df['trial8_target_policy_frozen'].map(as_bool).all()
    ),
    'multimodal_preflight_passed_for_all': bool(
        manifest_df['trial8_v2_multimodal_loader_preflight_passed']
        .map(as_bool).all()
    ),
    'source_manifest_training_is_locked': not bool(
        manifest_df['trial8_model_training_authorized'].map(as_bool).any()
    ),
}
if not all(manifest_checks.values()):
    failed = [name for name, passed in manifest_checks.items() if not passed]
    raise RuntimeError('V2.2 manifest gate failure: ' + ', '.join(failed))

shape_rows = []
observed_spatial_shapes = []
for case_index, row in enumerate(manifest_df.to_dict('records'), start=1):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    if not npz_path.exists():
        raise FileNotFoundError(f'{subject_id}: source NPZ does not exist.')
    observed_npz_hash = file_sha256(npz_path)
    if observed_npz_hash != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: frozen source NPZ hash changed.')
    image_shape, image_fortran, image_dtype = npz_array_header(npz_path, 'image')
    if len(image_shape) != 4 or image_shape[0] != 15:
        raise RuntimeError(
            f'{subject_id}: expected [15, X, Y, Z], observed {image_shape}.'
        )
    spatial_shape = tuple(int(value) for value in image_shape[1:])
    observed_spatial_shapes.append(spatial_shape)
    role = (
        'fold0_fit'
        if subject_id in set(fold0_fit_df['subject_id'].astype(str))
        else 'fold0_heldout'
        if subject_id in set(fold0_heldout_df['subject_id'].astype(str))
        else f'locked_{row["dev_split"]}'
    )
    shape_rows.append({
        'subject_id': subject_id,
        'dev_split': str(row['dev_split']),
        'development_fold': int(row['development_fold']),
        'block1_role': role,
        'image_channels': int(image_shape[0]),
        'shape_x': spatial_shape[0],
        'shape_y': spatial_shape[1],
        'shape_z': spatial_shape[2],
        'image_dtype': str(image_dtype),
        'fortran_order': image_fortran,
        'source_npz_sha256_verified': True,
        'target_arrays_read': False,
    })
    if case_index == 1 or case_index % 20 == 0 or case_index == len(manifest_df):
        print(
            f'Header/hash audit {case_index}/{len(manifest_df)}: '
            f'{subject_id}, shape={spatial_shape}'
        )

maximum_observed_shape = tuple(
    max(shape[axis] for shape in observed_spatial_shapes) for axis in range(3)
)
global_canvas_shape = tuple(
    round_up(maximum_observed_shape[axis], GLOBAL_CANVAS_DIVISOR[axis])
    for axis in range(3)
)
if any(
    global_canvas_shape[axis] > GLOBAL_CANVAS_LIMIT[axis]
    for axis in range(3)
):
    raise RuntimeError(
        'Observed shapes exceed the predeclared global-canvas limit: '
        f'observed max={maximum_observed_shape}, canvas={global_canvas_shape}, '
        f'limit={GLOBAL_CANVAS_LIMIT}.'
    )

for record in shape_rows:
    shape = (record['shape_x'], record['shape_y'], record['shape_z'])
    pad_total = [global_canvas_shape[i] - shape[i] for i in range(3)]
    pad_before = [value // 2 for value in pad_total]
    pad_after = [pad_total[i] - pad_before[i] for i in range(3)]
    for axis, name in enumerate('xyz'):
        record[f'pad_{name}_before'] = pad_before[axis]
        record[f'pad_{name}_after'] = pad_after[axis]
    record['global_canvas_shape'] = str(global_canvas_shape)
    record['global_input_shape'] = str(tuple(
        global_canvas_shape[i] // GLOBAL_IMAGE_DOWNSAMPLE[i]
        for i in range(3)
    ))
    record['target_arrays_read'] = False

shape_df = pd.DataFrame(shape_rows)
atomic_write_csv(shape_df, GLOBAL_SHAPE_AUDIT_PATH)






def physical_ball(spacing, radius_mm):
    radii = [int(math.ceil(radius_mm / value)) for value in spacing]
    grid = np.ogrid[
        tuple(slice(-radius, radius + 1) for radius in radii)
    ]
    distance_squared = np.zeros(
        tuple(2 * radius + 1 for radius in radii), dtype=np.float32
    )
    for axis, coordinates in enumerate(grid):
        distance_squared += (coordinates.astype(np.float32) * spacing[axis]) ** 2
    return distance_squared <= radius_mm ** 2 + 1e-6


def component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'


def unique_pool_voxels(coordinates, offset, factor):
    if len(coordinates) == 0:
        return 0
    pooled = (
        (coordinates + np.asarray(offset, dtype=np.int64)[None])
        // np.asarray(factor, dtype=np.int64)[None]
    )
    return int(np.unique(pooled, axis=0).shape[0])


fit_subject_rows = []
component_rows = []
component_records = defaultdict(list)
fit_subject_ids = set(fold0_fit_df['subject_id'].astype(str))
fold0_heldout_ids = set(fold0_heldout_df['subject_id'].astype(str))
locked_subject_ids = set(locked_df['subject_id'].astype(str))
target_read_subjects = set()

for case_index, row in enumerate(
    fold0_fit_df.sort_values('subject_id').to_dict('records'), start=1
):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        required_target_arrays = {
            'target_binary', 'target_soft', 'binary_loss_weight',
            'crop_mask', 'target_spacing', 'source_target_volume_ml',
        }
        missing = sorted(required_target_arrays - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: target arrays missing: {missing}'
            )
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float64).reshape(-1)
        binary_loss_weight = scalar(data, 'binary_loss_weight')
        source_target_volume_ml = scalar(data, 'source_target_volume_ml')
    target_read_subjects.add(subject_id)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target and crop shapes differ.')
    if binary.shape != tuple(
        shape_df.set_index('subject_id').loc[subject_id, [
            'shape_x', 'shape_y', 'shape_z'
        ]].astype(int)
    ):
        raise RuntimeError(f'{subject_id}: target and image shapes differ.')
    if spacing.shape != (3,) or not np.isfinite(spacing).all() or (spacing <= 0).any():
        raise RuntimeError(f'{subject_id}: target spacing is invalid.')
    if not np.isfinite(soft).all() or (soft < 0).any() or (soft > 1).any():
        raise RuntimeError(f'{subject_id}: soft target is invalid.')

    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        support = binary
        expected_hard_weight = 1.0
    elif supervision_mode == 'soft_only_subvoxel':
        support = soft > 0
        expected_hard_weight = 0.0
    elif supervision_mode == 'true_empty':
        support = np.zeros_like(binary)
        expected_hard_weight = 1.0
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    if not math.isclose(binary_loss_weight, expected_hard_weight):
        raise RuntimeError(f'{subject_id}: binary loss weight policy changed.')
    if as_bool(row['trial8_sampling_positive']) != bool(support.any()):
        raise RuntimeError(f'{subject_id}: sampling-positive policy mismatch.')

    connectivity = np.ones((3, 3, 3), dtype=bool)
    atomic_labels, atomic_count = ndimage.label(support, structure=connectivity)
    if support.any():
        dilation_structure = physical_ball(
            spacing, PROPOSAL_CLUSTER_DILATION_MM
        )
        dilated_support = ndimage.binary_dilation(
            support, structure=dilation_structure
        )
        cluster_labels, cluster_count = ndimage.label(
            dilated_support, structure=connectivity
        )
    else:
        cluster_labels = np.zeros_like(atomic_labels)
        cluster_count = 0

    shape = np.asarray(binary.shape, dtype=np.int64)
    pad_total = np.asarray(global_canvas_shape, dtype=np.int64) - shape
    pad_before = pad_total // 2
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    support_inside_crop = int((support & crop_mask).sum())
    subject_component_ids = []

    for atomic_id in range(1, atomic_count + 1):
        coordinates = np.argwhere(atomic_labels == atomic_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty labeled component.')
        cluster_values = cluster_labels[
            tuple(coordinates[:, axis] for axis in range(3))
        ]
        cluster_values = cluster_values[cluster_values > 0]
        if len(cluster_values) == 0:
            raise RuntimeError(f'{subject_id}: component lost during clustering.')
        proposal_cluster_id = int(
            np.bincount(cluster_values.astype(np.int64)).argmax()
        )
        component_voxels = int(len(coordinates))
        component_volume_ml = component_voxels * voxel_volume_ml
        minimum = coordinates.min(axis=0)
        maximum = coordinates.max(axis=0) + 1
        centroid_voxels = coordinates.mean(axis=0)
        soft_values = soft[
            tuple(coordinates[:, axis] for axis in range(3))
        ]
        global_proposal_voxels = unique_pool_voxels(
            coordinates, pad_before, GLOBAL_PROPOSAL_FACTOR
        )
        local_proposal_voxels = unique_pool_voxels(
            coordinates, (0, 0, 0), LOCAL_PROPOSAL_FACTOR
        )
        record = {
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'subject_volume_bin': str(
                row['trial8_effective_lesion_volume_bin']
            ),
            'atomic_component_id': atomic_id,
            'proposal_cluster_id': proposal_cluster_id,
            'component_tier': component_tier(
                component_volume_ml, component_voxels, voxel_volume_ml
            ),
            'component_voxels': component_voxels,
            'component_volume_ml': component_volume_ml,
            'component_soft_sum': float(soft_values.sum(dtype=np.float64)),
            'component_soft_max': float(soft_values.max(initial=0.0)),
            'centroid_x_vox': float(centroid_voxels[0]),
            'centroid_y_vox': float(centroid_voxels[1]),
            'centroid_z_vox': float(centroid_voxels[2]),
            'centroid_x_mm': float(centroid_voxels[0] * spacing[0]),
            'centroid_y_mm': float(centroid_voxels[1] * spacing[1]),
            'centroid_z_mm': float(centroid_voxels[2] * spacing[2]),
            'bbox_x0': int(minimum[0]),
            'bbox_y0': int(minimum[1]),
            'bbox_z0': int(minimum[2]),
            'bbox_x1': int(maximum[0]),
            'bbox_y1': int(maximum[1]),
            'bbox_z1': int(maximum[2]),
            'global_proposal_voxels': global_proposal_voxels,
            'local_proposal_voxels': local_proposal_voxels,
            'preserved_in_global_proposal': global_proposal_voxels > 0,
            'preserved_in_local_proposal': local_proposal_voxels > 0,
            'inside_crop_fraction': float(
                crop_mask[
                    tuple(coordinates[:, axis] for axis in range(3))
                ].mean()
            ),
            'outside_crop_voxels': int((
                ~crop_mask[
                    tuple(coordinates[:, axis] for axis in range(3))
                ]
            ).sum()),
            'touches_array_edge': bool(
                np.any(minimum == 0) or np.any(maximum == shape)
            ),
            'minimum_array_edge_distance_vox': int(np.min(np.concatenate([
                minimum,
                shape - maximum,
            ]))),
            'target_arrays_read': True,
        }
        component_rows.append(record)
        component_records[subject_id].append(record)
        subject_component_ids.append(atomic_id)

    fit_subject_rows.append({
        'subject_id': subject_id,
        'development_fold': int(row['development_fold']),
        'supervision_mode': supervision_mode,
        'subject_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
        'effective_target_volume_ml': float(
            row['trial8_effective_target_volume_ml']
        ),
        'source_target_volume_ml': source_target_volume_ml,
        'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
        'sampling_positive': as_bool(row['trial8_sampling_positive']),
        'atomic_components_26n': int(atomic_count),
        'proposal_clusters': int(cluster_count),
        'sampling_support_voxels': int(support.sum()),
        'binary_voxels': int(binary.sum()),
        'soft_nonzero_voxels': int((soft > 0).sum()),
        'soft_sum': float(soft.sum(dtype=np.float64)),
        'spacing_x_mm': float(spacing[0]),
        'spacing_y_mm': float(spacing[1]),
        'spacing_z_mm': float(spacing[2]),
        'voxel_volume_ml': voxel_volume_ml,
        'support_inside_crop_fraction': (
            1.0 if not support.any() else support_inside_crop / int(support.sum())
        ),
        'all_components_preserved_global': all(
            item['preserved_in_global_proposal']
            for item in component_records[subject_id]
        ),
        'all_components_preserved_local': all(
            item['preserved_in_local_proposal']
            for item in component_records[subject_id]
        ),
        'target_arrays_read': True,
    })
    if case_index == 1 or case_index % 10 == 0 or case_index == len(fold0_fit_df):
        print(
            f'Component audit {case_index}/{len(fold0_fit_df)}: '
            f'{subject_id}, components={atomic_count}, clusters={cluster_count}'
        )

fit_subject_df = pd.DataFrame(fit_subject_rows)
component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError('No fold-0 fitting components were found.')
atomic_write_csv(fit_subject_df, FIT_SUBJECT_AUDIT_PATH)
atomic_write_csv(component_df, COMPONENT_AUDIT_PATH)
mask_disagreement_df = component_df[
    component_df['outside_crop_voxels'].gt(0)
].copy().sort_values(
    ['outside_crop_voxels', 'component_volume_ml'], ascending=[False, False]
)
atomic_write_csv(mask_disagreement_df, COMPONENT_MASK_DISAGREEMENT_PATH)
atomic_write_npz(
    COMPONENT_INDEX_PATH,
    subject_id=component_df['subject_id'].astype(str).to_numpy(dtype='U32'),
    atomic_component_id=component_df['atomic_component_id'].to_numpy(
        dtype=np.int32
    ),
    proposal_cluster_id=component_df['proposal_cluster_id'].to_numpy(
        dtype=np.int32
    ),
    component_tier=component_df['component_tier'].astype(str).to_numpy(
        dtype='U32'
    ),
    component_voxels=component_df['component_voxels'].to_numpy(dtype=np.int32),
    component_volume_ml=component_df['component_volume_ml'].to_numpy(
        dtype=np.float32
    ),
    centroid_voxels=component_df[[
        'centroid_x_vox', 'centroid_y_vox', 'centroid_z_vox'
    ]].to_numpy(dtype=np.float32),
    bbox_start=component_df[[
        'bbox_x0', 'bbox_y0', 'bbox_z0'
    ]].to_numpy(dtype=np.int32),
    bbox_stop=component_df[[
        'bbox_x1', 'bbox_y1', 'bbox_z1'
    ]].to_numpy(dtype=np.int32),
)






fit_by_subject = {
    str(row['subject_id']): row
    for row in fold0_fit_df.to_dict('records')
}
positive_subjects = sorted(
    str(subject_id)
    for subject_id in fit_subject_df.loc[
        fit_subject_df['sampling_positive'], 'subject_id'
    ]
)


def interleaved_subjects_by_bin(subjects, rng):
    by_bin = defaultdict(list)
    for subject_id in subjects:
        lesion_bin = str(
            fit_by_subject[subject_id]['trial8_effective_lesion_volume_bin']
        )
        by_bin[lesion_bin].append(subject_id)
    for lesion_bin in by_bin:
        values = np.asarray(sorted(by_bin[lesion_bin]), dtype=object)
        by_bin[lesion_bin] = list(values[rng.permutation(len(values))])
    ordered_bins = sorted(by_bin)
    output = []
    cursor = 0
    while True:
        added = False
        for lesion_bin in ordered_bins:
            if cursor < len(by_bin[lesion_bin]):
                output.append(by_bin[lesion_bin][cursor])
                added = True
        if not added:
            break
        cursor += 1
    return output


def choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if not records:
        raise RuntimeError(f'{subject_id}: positive subject has no components.')
    mode = str(fit_by_subject[subject_id]['trial8_supervision_mode'])
    if mode == 'soft_only_subvoxel':
        index = (epoch + occurrence) % len(records)
        return records[index]
    volumes = np.asarray([
        max(record['component_volume_ml'], 1e-6) for record in records
    ], dtype=np.float64)
    weights = np.power(volumes, -0.35)
    if occurrence > 0 and len(records) > 1:
        first_index = int((epoch + len(subject_id)) % len(records))
        weights[first_index] = 0.0
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def build_schedule(epoch):
    rng = np.random.default_rng(SEED + int(epoch) * 10_007)
    rows = []
    subject_occurrence = defaultdict(int)
    base_subjects = interleaved_subjects_by_bin(positive_subjects, rng)
    for subject_id in base_subjects:
        component = choose_component(subject_id, epoch, 0, rng)
        rows.append({
            'epoch': int(epoch),
            'sample_index': len(rows),
            'sample_type': 'positive',
            'sample_subtype': 'positive_base',
            'subject_id': subject_id,
            'subject_volume_bin': str(
                fit_by_subject[subject_id][
                    'trial8_effective_lesion_volume_bin'
                ]
            ),
            'supervision_mode': str(
                fit_by_subject[subject_id]['trial8_supervision_mode']
            ),
            'atomic_component_id': component['atomic_component_id'],
            'proposal_cluster_id': component['proposal_cluster_id'],
            'component_tier': component['component_tier'],
            'component_volume_ml': component['component_volume_ml'],
            'target_arrays_read_for_schedule': False,
        })
        subject_occurrence[subject_id] += 1

    extras_needed = POSITIVE_PATCHES_PER_EPOCH - len(rows)
    extra_candidates = interleaved_subjects_by_bin(positive_subjects, rng)
    extra_subjects = []
    for subject_id in extra_candidates:
        if (
            subject_occurrence[subject_id]
            < MAX_POSITIVE_PATCHES_PER_SUBJECT_PER_EPOCH
        ):
            extra_subjects.append(subject_id)
        if len(extra_subjects) == extras_needed:
            break
    if len(extra_subjects) != extras_needed:
        raise RuntimeError('Could not allocate the positive schedule safely.')
    for subject_id in extra_subjects:
        occurrence = subject_occurrence[subject_id]
        component = choose_component(subject_id, epoch, occurrence, rng)
        rows.append({
            'epoch': int(epoch),
            'sample_index': len(rows),
            'sample_type': 'positive',
            'sample_subtype': 'positive_extra_bin_balanced',
            'subject_id': subject_id,
            'subject_volume_bin': str(
                fit_by_subject[subject_id][
                    'trial8_effective_lesion_volume_bin'
                ]
            ),
            'supervision_mode': str(
                fit_by_subject[subject_id]['trial8_supervision_mode']
            ),
            'atomic_component_id': component['atomic_component_id'],
            'proposal_cluster_id': component['proposal_cluster_id'],
            'component_tier': component['component_tier'],
            'component_volume_ml': component['component_volume_ml'],
            'target_arrays_read_for_schedule': False,
        })
        subject_occurrence[subject_id] += 1

    topology_eligible = sorted(
        subject_id for subject_id in fit_by_subject
        if as_bool(fit_by_subject[subject_id]['trial8_atlas_enabled'])
    )
    topology_order = list(np.asarray(topology_eligible, dtype=object)[
        rng.permutation(len(topology_eligible))
    ])
    for subject_id in topology_order[:TOPOLOGY_NEGATIVE_PATCHES_PER_EPOCH]:
        rows.append({
            'epoch': int(epoch),
            'sample_index': len(rows),
            'sample_type': 'topology_negative',
            'sample_subtype': 'arterial_neighbor_or_contralateral',
            'subject_id': subject_id,
            'subject_volume_bin': str(
                fit_by_subject[subject_id][
                    'trial8_effective_lesion_volume_bin'
                ]
            ),
            'supervision_mode': str(
                fit_by_subject[subject_id]['trial8_supervision_mode']
            ),
            'atomic_component_id': 0,
            'proposal_cluster_id': 0,
            'component_tier': 'none',
            'component_volume_ml': 0.0,
            'target_arrays_read_for_schedule': False,
        })

    all_fit_subjects = sorted(fit_by_subject)
    brain_order = list(np.asarray(all_fit_subjects, dtype=object)[
        rng.permutation(len(all_fit_subjects))
    ])
    for subject_id in brain_order[:BRAIN_NEGATIVE_PATCHES_PER_EPOCH]:
        rows.append({
            'epoch': int(epoch),
            'sample_index': len(rows),
            'sample_type': 'brain_negative',
            'sample_subtype': 'intracranial_target_excluded',
            'subject_id': subject_id,
            'subject_volume_bin': str(
                fit_by_subject[subject_id][
                    'trial8_effective_lesion_volume_bin'
                ]
            ),
            'supervision_mode': str(
                fit_by_subject[subject_id]['trial8_supervision_mode']
            ),
            'atomic_component_id': 0,
            'proposal_cluster_id': 0,
            'component_tier': 'none',
            'component_volume_ml': 0.0,
            'target_arrays_read_for_schedule': False,
        })
    return pd.DataFrame(rows)


schedule_frames = [build_schedule(epoch) for epoch in AUDIT_EPOCHS]
repeat_frames = [build_schedule(epoch) for epoch in AUDIT_EPOCHS]
schedule_df = pd.concat(schedule_frames, ignore_index=True)
repeat_df = pd.concat(repeat_frames, ignore_index=True)
schedule_deterministic = schedule_df.equals(repeat_df)
atomic_write_csv(schedule_df, SCHEDULE_PATH)

schedule_summary_df = schedule_df.groupby(
    ['epoch', 'sample_type'], as_index=False
).agg(
    patches=('subject_id', 'count'),
    subjects=('subject_id', 'nunique'),
    minimum_component_volume_ml=('component_volume_ml', 'min'),
    maximum_component_volume_ml=('component_volume_ml', 'max'),
)
atomic_write_csv(schedule_summary_df, SCHEDULE_SUMMARY_PATH)

soft_only_subjects = sorted(
    subject_id for subject_id in positive_subjects
    if str(fit_by_subject[subject_id]['trial8_supervision_mode'])
    == 'soft_only_subvoxel'
)
soft_only_alternates = True
for subject_id in soft_only_subjects:
    if len(component_records[subject_id]) > 1:
        selected = []
        for epoch in AUDIT_EPOCHS:
            selected.append(int(schedule_df[
                schedule_df['epoch'].eq(epoch)
                & schedule_df['subject_id'].eq(subject_id)
                & schedule_df['sample_subtype'].eq('positive_base')
            ]['atomic_component_id'].iloc[0]))
        soft_only_alternates = soft_only_alternates and len(set(selected)) == 2






with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as schema_data:
    schema = {name: np.asarray(schema_data[name]) for name in schema_data.files}
if any(array.dtype.kind == 'O' for array in schema.values()):
    raise RuntimeError('Verified graph schema unexpectedly contains objects.')

required_schema_arrays = {
    'node_labels', 'vessel_names', 'vessel_feature_names',
    'case_feature_names',
    'cross_edge_index', 'cross_edge_type', 'cross_edge_weight',
    'cross_relation_names',
}
missing_schema_arrays = sorted(required_schema_arrays - set(schema))
if missing_schema_arrays:
    raise RuntimeError(
        'Graph schema arrays missing: ' + ', '.join(missing_schema_arrays)
    )

node_labels = np.asarray(schema['node_labels'], dtype=np.int64)
vessel_names = decode_strings(schema['vessel_names'])
vessel_feature_names = decode_strings(schema['vessel_feature_names'])
case_feature_names = decode_strings(schema['case_feature_names'])
cross_edge_index = np.asarray(schema['cross_edge_index'], dtype=np.int64)
cross_edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
cross_edge_weight = np.asarray(schema['cross_edge_weight'], dtype=np.float32)
cross_relation_names = decode_strings(schema['cross_relation_names'])
risk_feature_indices = np.asarray([
    vessel_feature_names.index(name) for name in ARTERIAL_SOURCE_FEATURES
], dtype=np.int64)
required_confidence_features = [
    'atlas_enabled',
    'registration_confidence',
    'cow_prior_valid',
    'lvo_present',
    'perfusion_valid_fraction',
]
confidence_feature_indices = {
    name: case_feature_names.index(name)
    for name in required_confidence_features
}

topology_rows = []
for territory_index in range(30):
    edge_mask = cross_edge_index[1] == territory_index
    incoming_indices = np.where(edge_mask)[0]
    incoming_vessels = sorted(set(
        vessel_names[int(cross_edge_index[0, edge_index])]
        for edge_index in incoming_indices
    ))
    incoming_relations = sorted(set(
        cross_relation_names[int(cross_edge_type[edge_index])]
        for edge_index in incoming_indices
    ))
    topology_rows.append({
        'territory_index': territory_index,
        'territory_label': territory_index + 1,
        'incoming_edges': int(len(incoming_indices)),
        'incoming_vessels': '|'.join(incoming_vessels),
        'incoming_relations': '|'.join(incoming_relations),
        'has_primary_supply': 'primary_supply' in incoming_relations,
        'has_communicating_or_proximal_supply': (
            'communicating_or_proximal_supply' in incoming_relations
        ),
        'edge_direction': 'vessel_to_territory',
    })
topology_df = pd.DataFrame(topology_rows)
atomic_write_csv(topology_df, ARTERIAL_TOPOLOGY_PATH)

arterial_rows = []
all_sidecars_target_free = True
all_sidecar_hashes_match = True
for case_index, row in enumerate(manifest_df.to_dict('records'), start=1):
    subject_id = str(row['subject_id'])
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if not sidecar_path.exists():
        raise FileNotFoundError(f'{subject_id}: graph sidecar is missing.')
    sidecar_hash_matches = (
        file_sha256(sidecar_path) == str(row['trial8_graph_sidecar_sha256'])
    )
    all_sidecar_hashes_match = all_sidecar_hashes_match and sidecar_hash_matches
    if not sidecar_hash_matches:
        raise RuntimeError(f'{subject_id}: graph sidecar hash changed.')
    with np.load(sidecar_path, allow_pickle=False) as data:
        target_free = not bool(FORBIDDEN_SIDECAR_KEYS & set(data.files))
        all_sidecars_target_free = all_sidecars_target_free and target_free
        if not target_free:
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        vessel_features = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_feature_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_presence = np.asarray(data['vessel_presence']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(data['node_reliability'], dtype=np.float32)
        case_features = np.asarray(data['case_features'], dtype=np.float32)
    if vessel_features.shape != (13, 16):
        raise RuntimeError(f'{subject_id}: vessel feature shape changed.')
    if vessel_feature_valid.shape != (13, 16):
        raise RuntimeError(f'{subject_id}: vessel validity shape changed.')
    if vessel_presence.shape != (13,) or node_valid.shape != (30,):
        raise RuntimeError(f'{subject_id}: graph node validity shape changed.')
    if node_reliability.shape != (30,):
        raise RuntimeError(f'{subject_id}: territory reliability shape changed.')
    if case_features.shape != (11,):
        raise RuntimeError(f'{subject_id}: case feature shape changed.')
    if not np.isfinite(vessel_features).all() or not np.isfinite(
        node_reliability
    ).all() or not np.isfinite(case_features).all():
        raise RuntimeError(f'{subject_id}: graph sidecar is non-finite.')

    vessel_evidence_valid = (
        vessel_presence
        & vessel_feature_valid[:, risk_feature_indices].any(axis=1)
    )
    territory_projection_valid = np.zeros(30, dtype=bool)
    territory_valid_source_count = np.zeros(30, dtype=np.int16)
    for edge_index in range(cross_edge_index.shape[1]):
        vessel_index = int(cross_edge_index[0, edge_index])
        territory_index = int(cross_edge_index[1, edge_index])
        if vessel_evidence_valid[vessel_index]:
            territory_valid_source_count[territory_index] += 1
    territory_projection_valid = (
        territory_valid_source_count > 0
    ) & node_valid & (node_reliability > 0)

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    registration_confidence = float(case_features[
        confidence_feature_indices['registration_confidence']
    ])
    perfusion_valid_fraction = float(case_features[
        confidence_feature_indices['perfusion_valid_fraction']
    ])
    cow_prior_valid = bool(case_features[
        confidence_feature_indices['cow_prior_valid']
    ] > 0.5)
    lvo_present = bool(case_features[
        confidence_feature_indices['lvo_present']
    ] > 0.5)
    sidecar_atlas_enabled = bool(case_features[
        confidence_feature_indices['atlas_enabled']
    ] > 0.5)
    projection_confidence = (
        registration_confidence * perfusion_valid_fraction
        if atlas_enabled
        else 0.0
    )
    projected_voxels = 0
    atlas_voxels = 0
    region_labels_valid = True
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        if not atlas_path.exists():
            raise FileNotFoundError(f'{subject_id}: atlas map is missing.')
        atlas_image = nib.load(str(atlas_path))
        region_map = np.asanyarray(atlas_image.dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int16)
        expected_shape = tuple(
            shape_df.set_index('subject_id').loc[
                subject_id, ['shape_x', 'shape_y', 'shape_z']
            ].astype(int)
        )
        if region_map.shape != expected_shape:
            raise RuntimeError(
                f'{subject_id}: atlas/image shape mismatch: '
                f'{region_map.shape} vs {expected_shape}.'
            )
        region_labels_valid = bool(
            region_map.min(initial=0) >= 0 and region_map.max(initial=0) <= 30
        )
        if not region_labels_valid:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
        atlas_voxels = int((region_map > 0).sum())
        projection_lookup = np.concatenate([
            np.asarray([False]), territory_projection_valid
        ])
        projected_voxels = int(projection_lookup[region_map].sum())
    else:
        territory_projection_valid[:] = False
        territory_valid_source_count[:] = 0

    projectable_source_counts = territory_valid_source_count[
        territory_projection_valid
    ]
    arterial_rows.append({
        'subject_id': subject_id,
        'dev_split': str(row['dev_split']),
        'development_fold': int(row['development_fold']),
        'atlas_enabled': atlas_enabled,
        'sidecar_atlas_enabled': sidecar_atlas_enabled,
        'atlas_enabled_matches_sidecar': atlas_enabled == sidecar_atlas_enabled,
        'registration_confidence': registration_confidence,
        'perfusion_valid_fraction': perfusion_valid_fraction,
        'projection_confidence': projection_confidence,
        'cow_prior_valid': cow_prior_valid,
        'lvo_present': lvo_present,
        'valid_vessel_nodes': int(vessel_presence.sum()),
        'vessels_with_arterial_evidence': int(vessel_evidence_valid.sum()),
        'valid_territory_nodes': int(node_valid.sum()),
        'projectable_territory_nodes': int(territory_projection_valid.sum()),
        'minimum_valid_sources_per_projectable_territory': int(
            projectable_source_counts.min()
            if len(projectable_source_counts)
            else 0
        ),
        'maximum_valid_sources_per_territory': int(
            territory_valid_source_count.max(initial=0)
        ),
        'atlas_label_voxels': atlas_voxels,
        'projection_valid_voxels': projected_voxels,
        'projection_valid_atlas_fraction': (
            projected_voxels / atlas_voxels if atlas_voxels else 0.0
        ),
        'region_labels_valid': region_labels_valid,
        'disabled_projection_is_zero': (
            True if atlas_enabled else projected_voxels == 0
        ),
        'graph_sidecar_target_free': target_free,
        'graph_sidecar_sha256_verified': sidecar_hash_matches,
        'target_arrays_read': False,
    })
    if case_index == 1 or case_index % 20 == 0 or case_index == len(manifest_df):
        print(
            f'Arterial audit {case_index}/{len(manifest_df)}: '
            f'{subject_id}, projectable={int(territory_projection_valid.sum())}'
        )

arterial_df = pd.DataFrame(arterial_rows)
atomic_write_csv(arterial_df, ARTERIAL_AUDIT_PATH)






contract_record = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    **ARCHITECTURE_CONTRACT,
    'frozen_global_canvas_shape': list(global_canvas_shape),
    'frozen_global_input_shape': [
        global_canvas_shape[i] // GLOBAL_IMAGE_DOWNSAMPLE[i]
        for i in range(3)
    ],
    'maximum_observed_native_shape': list(maximum_observed_shape),
    'fold0_fit_subject_hash': subject_hash(fit_subject_ids),
    'fold0_heldout_subject_hash': subject_hash(fold0_heldout_ids),
    'locked_subject_hash': subject_hash(locked_subject_ids),
    'input_sha256': observed_hashes,
    'architecture_contract_sha256': CONTRACT_SHA256,
    
    
    
    'architecture_implementation_authorized': False,
    'pilot_training_authorized': False,
}
atomic_write_json(contract_record, ARCHITECTURE_CONTRACT_PATH)

epoch_counts = schedule_df.groupby(['epoch', 'sample_type']).size().unstack(
    fill_value=0
)
positive_per_subject = schedule_df[
    schedule_df['sample_type'].eq('positive')
].groupby(['epoch', 'subject_id']).size()

component_checks = {
    'targets_read_for_exactly_82_fold0_fit_subjects': (
        target_read_subjects == fit_subject_ids
        and len(target_read_subjects) == EXPECTED_FOLD0_FIT
    ),
    'fold0_heldout_targets_not_read': not bool(
        target_read_subjects & fold0_heldout_ids
    ),
    'locked_validation_and_test_targets_not_read': not bool(
        target_read_subjects & locked_subject_ids
    ),
    'fit_subject_audit_has_82_rows': len(fit_subject_df) == EXPECTED_FOLD0_FIT,
    'exactly_one_soft_only_subject_is_present': int(
        fit_subject_df['supervision_mode'].eq('soft_only_subvoxel').sum()
    ) == 1,
    'at_most_one_true_empty_subject_is_present': int(
        fit_subject_df['supervision_mode'].eq('true_empty').sum()
    ) <= 1,
    'soft_only_subject_has_multiple_atomic_components': bool(
        fit_subject_df.loc[
            fit_subject_df['supervision_mode'].eq('soft_only_subvoxel'),
            'atomic_components_26n',
        ].ge(2).all()
    ),
    'positive_subjects_have_components': bool(
        fit_subject_df.loc[
            fit_subject_df['sampling_positive'], 'atomic_components_26n'
        ].gt(0).all()
    ),
    'true_empty_subjects_have_no_components': bool(
        fit_subject_df.loc[
            fit_subject_df['supervision_mode'].eq('true_empty'),
            'atomic_components_26n',
        ].eq(0).all()
    ),
    'component_sampling_mask_overlap_fractions_are_valid': bool(
        component_df['inside_crop_fraction'].between(0.0, 1.0).all()
    ),
    'sampling_mask_disagreement_is_explicitly_audited': (
        'outside_crop_voxels' in component_df.columns
        and bool(component_df['outside_crop_voxels'].ge(0).all())
    ),
    'no_components_touch_final_array_edges': not bool(
        component_df['touches_array_edge'].any()
    ),
    'all_components_preserved_in_global_proposal': bool(
        component_df['preserved_in_global_proposal'].all()
    ),
    'all_components_preserved_in_local_proposal': bool(
        component_df['preserved_in_local_proposal'].all()
    ),
    'proposal_clusters_do_not_exceed_atomic_components': bool(
        fit_subject_df['proposal_clusters'].le(
            fit_subject_df['atomic_components_26n']
        ).all()
    ),
    'component_index_is_pickle_free': True,
}
with np.load(COMPONENT_INDEX_PATH, allow_pickle=False) as component_index:
    component_checks['component_index_is_pickle_free'] = not any(
        np.asarray(component_index[name]).dtype.kind == 'O'
        for name in component_index.files
    )

schedule_checks = {
    'two_schedule_epochs_audited': set(schedule_df['epoch']) == set(AUDIT_EPOCHS),
    'schedule_is_deterministic': schedule_deterministic,
    'each_epoch_has_112_positive_patches': bool(
        epoch_counts['positive'].eq(POSITIVE_PATCHES_PER_EPOCH).all()
    ),
    'each_epoch_has_28_topology_negatives': bool(
        epoch_counts['topology_negative'].eq(
            TOPOLOGY_NEGATIVE_PATCHES_PER_EPOCH
        ).all()
    ),
    'each_epoch_has_28_brain_negatives': bool(
        epoch_counts['brain_negative'].eq(
            BRAIN_NEGATIVE_PATCHES_PER_EPOCH
        ).all()
    ),
    'each_epoch_has_168_total_samples': bool(
        schedule_df.groupby('epoch').size().eq(168).all()
    ),
    'positive_fraction_is_two_thirds': all(
        math.isclose(
            epoch_counts.loc[epoch, 'positive']
            / schedule_df[schedule_df['epoch'].eq(epoch)].shape[0],
            2.0 / 3.0,
        )
        for epoch in AUDIT_EPOCHS
    ),
    'all_positive_subjects_covered_each_epoch': all(
        set(schedule_df[
            schedule_df['epoch'].eq(epoch)
            & schedule_df['sample_type'].eq('positive')
        ]['subject_id']) == set(positive_subjects)
        for epoch in AUDIT_EPOCHS
    ),
    'per_subject_positive_cap_respected': bool(
        positive_per_subject.le(
            MAX_POSITIVE_PATCHES_PER_SUBJECT_PER_EPOCH
        ).all()
    ),
    'soft_only_components_alternate': soft_only_alternates,
    'schedule_contains_only_fold0_fit_subjects': set(
        schedule_df['subject_id'].astype(str)
    ).issubset(fit_subject_ids),
    'schedule_does_not_reopen_target_arrays': not bool(
        schedule_df['target_arrays_read_for_schedule'].any()
    ),
}

shape_checks = {
    'all_148_source_npz_hashes_verified': bool(
        shape_df['source_npz_sha256_verified'].all()
    ),
    'all_images_have_15_channels': bool(shape_df['image_channels'].eq(15).all()),
    'global_canvas_contains_every_native_shape': all(
        all(shape[axis] <= global_canvas_shape[axis] for axis in range(3))
        for shape in observed_spatial_shapes
    ),
    'global_canvas_within_predeclared_limit': all(
        global_canvas_shape[axis] <= GLOBAL_CANVAS_LIMIT[axis]
        for axis in range(3)
    ),
    'global_canvas_divisible_by_32': all(
        global_canvas_shape[axis] % GLOBAL_CANVAS_DIVISOR[axis] == 0
        for axis in range(3)
    ),
    'shape_audit_read_no_target_arrays': not bool(
        shape_df['target_arrays_read'].any()
    ),
}

arterial_checks = {
    'graph_schema_has_30_territories': np.array_equal(
        node_labels, np.arange(1, 31, dtype=np.int64)
    ),
    'graph_schema_has_13_vessels': len(vessel_names) == 13,
    'cross_relations_match_frozen_schema': (
        cross_relation_names == EXPECTED_CROSS_RELATIONS
    ),
    'cross_edge_index_shape_is_2_by_68': cross_edge_index.shape == (2, 68),
    'cross_source_indices_are_vessels': bool(
        cross_edge_index[0].min() >= 0
        and cross_edge_index[0].max() < 13
    ),
    'cross_target_indices_are_territories': bool(
        cross_edge_index[1].min() >= 0
        and cross_edge_index[1].max() < 30
    ),
    'cross_edge_weights_are_finite_positive': bool(
        np.isfinite(cross_edge_weight).all()
        and (cross_edge_weight > 0).all()
    ),
    'all_territories_have_incoming_supply': bool(
        topology_df['incoming_edges'].gt(0).all()
    ),
    'all_territories_have_primary_supply': bool(
        topology_df['has_primary_supply'].all()
    ),
    'all_arterial_source_features_exist': set(
        ARTERIAL_SOURCE_FEATURES
    ).issubset(vessel_feature_names),
    'all_confidence_features_exist': set(
        required_confidence_features
    ).issubset(case_feature_names),
    'all_148_sidecars_hash_verified': all_sidecar_hashes_match,
    'all_148_sidecars_target_free': all_sidecars_target_free,
    'all_region_labels_valid': bool(arterial_df['region_labels_valid'].all()),
    'manifest_and_sidecar_atlas_flags_match': bool(
        arterial_df['atlas_enabled_matches_sidecar'].all()
    ),
    'projection_confidence_is_finite_unit_interval': bool(
        np.isfinite(arterial_df['projection_confidence']).all()
        and arterial_df['projection_confidence'].between(0.0, 1.0).all()
    ),
    'atlas_disabled_projection_is_zero': bool(
        arterial_df.loc[
            ~arterial_df['atlas_enabled'], 'projectable_territory_nodes'
        ].eq(0).all()
    ),
    'at_least_one_enabled_case_has_projectable_territories': bool(
        arterial_df.loc[
            arterial_df['atlas_enabled'], 'projectable_territory_nodes'
        ].gt(0).any()
    ),
    'arterial_audit_read_no_target_arrays': not bool(
        arterial_df['target_arrays_read'].any()
    ),
}

contract_checks = {
    'contract_hash_is_stable': (
        canonical_json_sha256(ARCHITECTURE_CONTRACT) == CONTRACT_SHA256
    ),
    'reference_v2_path_is_preserved': (
        ARCHITECTURE_CONTRACT['reference_path']['reference_parameters']
        == EXPECTED_V2_PARAMETERS
    ),
    'global_to_local_path_is_required': bool(
        ARCHITECTURE_CONTRACT['global_to_local_design']
    ),
    'component_mil_is_required': 'component_mil_rule' in (
        ARCHITECTURE_CONTRACT['component_supervision']
    ),
    'arterial_projection_is_directed': (
        ARCHITECTURE_CONTRACT['directed_arterial_projection'][
            'edge_direction'
        ] == 'vessel node -> vascular territory node'
    ),
    'graph_cannot_write_direct_lesion_logits': (
        ARCHITECTURE_CONTRACT['directed_arterial_projection'][
            'direct_graph_to_lesion_logit'
        ] is False
    ),
    'clinical_branch_cannot_localize_directly': (
        ARCHITECTURE_CONTRACT['clinical_role'][
            'spatial_localization_allowed'
        ] is False
        and ARCHITECTURE_CONTRACT['clinical_role'][
            'direct_lesion_logit_allowed'
        ] is False
    ),
    'post_treatment_fields_are_excluded': (
        ARCHITECTURE_CONTRACT['clinical_role'][
            'post_imaging_treatment_fields_allowed'
        ] is False
        and ARCHITECTURE_CONTRACT['clinical_role'][
            'post_treatment_outcomes_allowed'
        ] is False
    ),
    'parameter_budget_is_bounded': (
        ARCHITECTURE_CONTRACT['parameter_budget'][
            'maximum_new_parameters'
        ] <= 2_500_000
    ),
    'mechanism_gate_is_not_a_sota_claim': 'not a SOTA claim' in (
        ARCHITECTURE_CONTRACT['predeclared_fold0_mechanism_gate'][
            'interpretation'
        ]
    ),
    'architecture_contract_written': ARCHITECTURE_CONTRACT_PATH.exists(),
}

leakage_checks = {
    'target_access_is_exactly_fold0_fit': target_read_subjects == fit_subject_ids,
    'fold0_heldout_target_arrays_not_read': True,
    'locked_validation_target_arrays_not_read': True,
    'test_target_arrays_not_read': True,
    'outcome_workbooks_not_read': True,
    'study_model_not_constructed': True,
    'training_not_performed': True,
    'optimizer_steps_are_zero': True,
    'study_trainable_parameters_are_zero': True,
    'pilot_training_remains_locked': True,
}

all_checks = {
    **upstream_checks,
    **manifest_checks,
    **shape_checks,
    **component_checks,
    **schedule_checks,
    **arterial_checks,
    **contract_checks,
    **leakage_checks,
}
preflight_passed = bool(all(bool(value) for value in all_checks.values()))



contract_record['architecture_implementation_authorized'] = preflight_passed
atomic_write_json(contract_record, ARCHITECTURE_CONTRACT_PATH)

audit_rows = []
check_groups = {
    'upstream': upstream_checks,
    'manifest': manifest_checks,
    'shape': shape_checks,
    'component': component_checks,
    'schedule': schedule_checks,
    'arterial': arterial_checks,
    'contract': contract_checks,
    'leakage': leakage_checks,
}
for category, checks in check_groups.items():
    for name, passed in checks.items():
        audit_rows.append({
            'category': category,
            'check': name,
            'passed': bool(passed),
        })
audit_df = pd.DataFrame(audit_rows)
atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)

output_paths = [
    FIT_SUBJECT_AUDIT_PATH,
    COMPONENT_AUDIT_PATH,
    COMPONENT_MASK_DISAGREEMENT_PATH,
    COMPONENT_INDEX_PATH,
    SCHEDULE_PATH,
    SCHEDULE_SUMMARY_PATH,
    GLOBAL_SHAPE_AUDIT_PATH,
    ARTERIAL_AUDIT_PATH,
    ARTERIAL_TOPOLOGY_PATH,
    ARCHITECTURE_CONTRACT_PATH,
    PREFLIGHT_AUDIT_PATH,
]
output_sha256 = {
    str(path): file_sha256(path) for path in output_paths if path.exists()
}

gate = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_version': MODEL_VERSION,
    'preflight_version': PREFLIGHT_VERSION,
    'code_revision': CODE_REVISION,
    'preflight_passed': preflight_passed,
    'checks': all_checks,
    'input_sha256': observed_hashes,
    'architecture_contract_path': str(ARCHITECTURE_CONTRACT_PATH),
    'architecture_contract_sha256': CONTRACT_SHA256,
    'output_sha256': output_sha256,
    'fold0_fit_subjects': len(fit_subject_ids),
    'fold0_heldout_subjects': len(fold0_heldout_ids),
    'fit_target_subject_hash': subject_hash(target_read_subjects),
    'fold0_fit_subject_hash': subject_hash(fit_subject_ids),
    'components_audited': len(component_df),
    'global_canvas_shape': list(global_canvas_shape),
    'global_input_shape': [
        global_canvas_shape[i] // GLOBAL_IMAGE_DOWNSAMPLE[i]
        for i in range(3)
    ],
    'directed_cross_edges': int(cross_edge_index.shape[1]),
    'target_arrays_read_splits': ['train_fold0_fit'],
    'fold0_heldout_targets_read': False,
    'locked_validation_targets_read': False,
    'test_targets_read': False,
    'outcome_workbooks_read': False,
    'study_model_constructed': False,
    'training_performed': False,
    'optimizer_steps': 0,
    'study_trainable_parameters': 0,
    'architecture_implementation_authorized': preflight_passed,
    'pilot_training_authorized': False,
    'full_fivefold_training_authorized': False,
    'next_block': (
        'construct and smoke-test VascuFateNet-v2.2 global proposal, '
        'directed arterial projection, and proposal-conditioned local '
        'refinement with neutrality, gradient, ablation, and L4-memory gates'
    ),
}
atomic_write_json(gate, PREFLIGHT_GATE_PATH)

print('\nFold-0 fit component summary:')
display(component_df.groupby('component_tier', as_index=False).agg(
    components=('atomic_component_id', 'count'),
    subjects=('subject_id', 'nunique'),
    minimum_volume_ml=('component_volume_ml', 'min'),
    median_volume_ml=('component_volume_ml', 'median'),
    maximum_volume_ml=('component_volume_ml', 'max'),
))
print('\nSampling-mask disagreement diagnostic:')
display(pd.DataFrame([{
    'components_total': len(component_df),
    'components_with_any_disagreement': len(mask_disagreement_df),
    'components_entirely_outside_sampling_mask': int(
        component_df['inside_crop_fraction'].eq(0.0).sum()
    ),
    'subjects_with_any_disagreement': int(
        mask_disagreement_df['subject_id'].nunique()
    ),
    'target_voxels_outside_sampling_mask': int(
        component_df['outside_crop_voxels'].sum()
    ),
    'components_touching_final_array_edge': int(
        component_df['touches_array_edge'].sum()
    ),
}]))
if len(mask_disagreement_df):
    display(mask_disagreement_df[[
        'subject_id', 'atomic_component_id', 'component_tier',
        'component_voxels', 'outside_crop_voxels', 'inside_crop_fraction',
        'touches_array_edge', 'minimum_array_edge_distance_vox',
    ]].head(20))
print('\nTwo-epoch component-balanced schedule:')
display(schedule_summary_df)
print('\nGlobal-view shape contract:')
display(pd.DataFrame([{
    'subjects': len(shape_df),
    'maximum_native_shape': str(maximum_observed_shape),
    'global_canvas_shape': str(global_canvas_shape),
    'global_input_shape': str(tuple(
        global_canvas_shape[i] // GLOBAL_IMAGE_DOWNSAMPLE[i]
        for i in range(3)
    )),
    'local_patch_shape': str(LOCAL_PATCH_SIZE),
}]))
print('\nDirected arterial projection by split:')
display(arterial_df.groupby('dev_split', as_index=False).agg(
    subjects=('subject_id', 'count'),
    atlas_enabled=('atlas_enabled', 'sum'),
    mean_valid_vessels=('valid_vessel_nodes', 'mean'),
    mean_projectable_territories=('projectable_territory_nodes', 'mean'),
    mean_projection_fraction=('projection_valid_atlas_fraction', 'mean'),
))
print('\nBlock 1 preflight checks:')
display(audit_df)

if not preflight_passed:
    failed = [name for name, passed in all_checks.items() if not bool(passed)]
    raise RuntimeError(
        'TRIAL8 V2.2 GLOBAL-TO-LOCAL BLOCK 1 PREFLIGHT FAILED. Do not '
        'construct or train V2.2. Failed checks: ' + ', '.join(failed)
    )

print('\n' + '=' * 80)
print('TRIAL8 V2.2 GLOBAL-TO-LOCAL COMPONENT/ARTERIAL PREFLIGHT PASSED')
print('=' * 80)
print(f'Fold-0 fit targets read:             {len(target_read_subjects)}')
print('Fold-0 held-out targets read:        False')
print('Locked validation targets read:      False')
print('Test targets read:                   False')
print(f'Atomic components audited:           {len(component_df)}')
print(f'Global canvas:                       {global_canvas_shape}')
print(
    'Global low-resolution input:          '
    f'{tuple(global_canvas_shape[i] // GLOBAL_IMAGE_DOWNSAMPLE[i] for i in range(3))}'
)
print(f'Directed vessel-to-territory edges:  {cross_edge_index.shape[1]}')
print('Study trainable parameters:          0 (architecture preflight only)')
print('Architecture implementation allowed: True')
print('Pilot training authorized:           False')
print(f'Architecture contract:               {ARCHITECTURE_CONTRACT_PATH}')
print(f'Preflight gate:                      {PREFLIGHT_GATE_PATH}')
print('\nNext: construct and smoke-test the V2.2 global-to-local model; do not train yet.')


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_negative_pool_exhaustive_skip_safe'
PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r2'
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 12
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch_cpu'])
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all(state['torch_cuda'])


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            positive_dice.append(
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            for component_id in range(1, count + 1):
                component = labels == component_id
                voxels = int(component.sum())
                tier = pilot_component_tier(voxels * voxel_volume_ml, voxels)
                detected = bool(prediction[component].any())
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    selection_score = (
        0.50 * mean_dice + 0.20 * lesion_f1
        + 0.15 * component_recall + 0.15 * small_recall
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['proposal_selection_score'] = (
        0.65 * proposal_threshold_df['component_recall']
        + 0.20 * proposal_threshold_df['positive_global_dice']
        + 0.15 * proposal_threshold_df['lesion_precision']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            for component_id in range(1, count + 1):
                component = labels == component_id
                voxels = int(component.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(probability[component].max(initial=0.0))
                proposal_max = float(proposal[component].max(initial=0.0))
                final_detected = final_max >= local_threshold
                proposal_covered = proposal_max >= proposal_threshold
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.75 * float(selected['selection_score'])
        + 0.15 * float(proposal_selected['component_recall'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and PILOT_BEST_CHECKPOINT_PATH.exists()
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.75*(0.50*mean_dice+0.20*lesion_f1+'
            '0.15*component_recall+0.15*small_component_recall)+'
            '0.15*proposal_component_recall+0.10*proposal_global_dice'
        ),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 ISOLATED FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        improved = current_score > best_score + EARLY_STOPPING_MINIMUM_DELTA
        if improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'improved': improved,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 ISOLATED FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


run_v22_fold0_pilot()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r4_device_safe_rng_checkpoint_migration'
PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r4'
PARENT_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r2'
PARENT_PILOT_CODE_REVISION = 'r2_negative_pool_exhaustive_skip_safe'
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PARENT_PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PARENT_PILOT_VERSION
)
PARENT_LAST_CHECKPOINT_PATH = PARENT_PILOT_ROOT / 'last_checkpoint.pt'
PARENT_HISTORY_PATH = PARENT_PILOT_ROOT / 'training_history.csv'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and PILOT_BEST_CHECKPOINT_PATH.exists()
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'parent_pilot_version': PARENT_PILOT_VERSION,
        'parent_last_checkpoint_path': str(PARENT_LAST_CHECKPOINT_PATH),
        'parent_last_checkpoint_sha256': (
            file_sha256(PARENT_LAST_CHECKPOINT_PATH)
            if PARENT_LAST_CHECKPOINT_PATH.exists() else None
        ),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    elif PARENT_LAST_CHECKPOINT_PATH.exists():
        parent = torch.load(
            PARENT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        parent_checks = {
            'pilot_version': (
                parent.get('pilot_version') == PARENT_PILOT_VERSION
            ),
            'code_revision': (
                parent.get('code_revision') == PARENT_PILOT_CODE_REVISION
            ),
            'parameter_count': int(parent.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'completed_at_least_one_epoch': int(parent.get('epoch', 0)) >= 1,
            'did_not_exceed_maximum_epochs': int(parent.get('epoch', 0))
            < MAX_EPOCHS,
        }
        if not all(parent_checks.values()):
            failed = [k for k, v in parent_checks.items() if not v]
            raise RuntimeError(
                'Unsafe r2-to-r4 checkpoint migration: ' + ', '.join(failed)
            )
        model.load_state_dict(parent['model_state'])
        ema_model.load_state_dict(parent['ema_model_state'])
        optimizer.load_state_dict(parent['optimizer_state'])
        scheduler.load_state_dict(parent['scheduler_state'])
        pilot_restore_rng_state(parent['rng_state'])
        migrated_epoch = int(parent['epoch'])
        start_epoch = migrated_epoch + 1
        update_step = int(parent['update_step'])
        validations_without_improvement = 0
        print('\nRecalibrating the r2 epoch-' + str(migrated_epoch)
              + ' last checkpoint under the repaired Dice/IoU metrics...')
        migrated_summary, migrated_tiers = pilot_validate(
            ema_model, migrated_epoch, device
        )
        best_score = float(migrated_summary['pilot_checkpoint_score'])
        best_epoch = migrated_epoch
        best_threshold = float(migrated_summary['threshold'])
        best_proposal_threshold = float(
            migrated_summary['proposal_threshold']
        )
        history_rows = [{
            'epoch': migrated_epoch,
            'migrated_from_pilot': PARENT_PILOT_VERSION,
            'validation_mean_dice': migrated_summary[
                'positive_mean_dice'
            ],
            'validation_median_dice': migrated_summary[
                'positive_median_dice'
            ],
            'validation_global_dice': migrated_summary[
                'positive_global_dice'
            ],
            'validation_lesion_f1': migrated_summary['lesion_f1'],
            'validation_lesion_recall': migrated_summary['lesion_recall'],
            'validation_component_recall': migrated_summary[
                'component_recall'
            ],
            'validation_small_component_recall': migrated_summary[
                'small_component_recall'
            ],
            'proposal_component_recall': migrated_summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': migrated_summary[
                'proposal_global_dice'
            ],
            'local_threshold': migrated_summary['threshold'],
            'proposal_threshold': migrated_summary['proposal_threshold'],
            'pilot_checkpoint_score': best_score,
            'best_score': best_score, 'best_epoch': best_epoch,
            'improved': True, 'validations_without_improvement': 0,
        }]
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        migrated_checkpoint = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'epoch': migrated_epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': 0,
            'validation_summary': migrated_summary,
            'migrated_from_checkpoint': str(PARENT_LAST_CHECKPOINT_PATH),
            'migrated_from_checkpoint_sha256': file_sha256(
                PARENT_LAST_CHECKPOINT_PATH
            ),
        }
        pilot_atomic_torch_save(
            migrated_checkpoint, PILOT_LAST_CHECKPOINT_PATH
        )
        pilot_atomic_torch_save(
            migrated_checkpoint, PILOT_BEST_CHECKPOINT_PATH
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 ISOLATED FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        improved = current_score > best_score + EARLY_STOPPING_MINIMUM_DELTA
        if improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'improved': improved,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 ISOLATED FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


run_v22_fold0_pilot()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r5_last_checkpoint_only',
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r5 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 ISOLATED FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        improved = current_score > best_score + EARLY_STOPPING_MINIMUM_DELTA
        if improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'improved': improved,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 ISOLATED FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


run_v22_fold0_pilot()


In [ ]:
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage


FUSION_DIAGNOSTIC_VERSION = 'v1_all20_best_vs_last_fixed_threshold'
FUSION_DIAGNOSTIC_CODE_REVISION = 'r3_positive_read_only_gate_assertions'
EXPECTED_R5_PILOT_VERSION = (
    'v2_2_fold0_global_local_component_diagnostic_r5'
)
EXPECTED_R5_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'

FUSION_BOOTSTRAP_REQUIRED = (
    'PROJECT_ROOT', 'MODEL_VERSION', 'PILOT_VERSION',
    'PILOT_CODE_REVISION', 'EXPECTED_V22_PARAMETERS',
    'VascuFateNetV22', 'pilot_schema', 'multimodal_gate',
    'fold0_heldout_df', 'heldout_rows', 'pilot_load_validation_case',
    'pilot_stack_sample', 'pilot_global_cache', 'pilot_local_from_cache',
    'pilot_sliding_starts', 'pilot_window_sample', 'PILOT_WINDOW_WEIGHT',
    'GLOBAL_CANVAS_SHAPE', 'LOCAL_PATCH_SIZE', 'pilot_lesion_counts',
    'pilot_target_component_iou_records', 'pilot_component_tier',
    'sample_global_to_local', 'symmetric_pad',
    'atomic_write_csv', 'atomic_write_json',
    'file_sha256', 'load_json', 'as_bool',
)
FUSION_BOOTSTRAP_MISSING = [
    name for name in FUSION_BOOTSTRAP_REQUIRED if name not in globals()
]
if FUSION_BOOTSTRAP_MISSING:
    raise RuntimeError(
        'TRIAL8 R5 FUSION DIAGNOSTIC RUNTIME BOOTSTRAP REQUIRED. '
        'This Colab runtime does not contain the completed r5 training-cell '
        'definitions, usually because the runtime was reset. Run the complete '
        'current trial8_v2_2_fold0_pilot_training.py block once in this '
        'runtime. It should recognize the completed r5 result gate, print '
        '"TRIAL8 V2.2 FOLD-0 PILOT ALREADY COMPLETE", and return without '
        'retraining or rewriting model checkpoints. Then rerun this diagnostic. '
        'Do not delete the existing r5 directory or checkpoints. Missing '
        'runtime symbols: ' + ', '.join(FUSION_BOOTSTRAP_MISSING)
    )

R5_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / EXPECTED_R5_PILOT_VERSION
)
R5_BEST_CHECKPOINT_PATH = R5_ROOT / 'best_checkpoint.pt'
R5_LAST_CHECKPOINT_PATH = R5_ROOT / 'last_checkpoint.pt'
R5_RESULT_GATE_PATH = R5_ROOT / 'pilot_result_gate.json'
R5_HISTORY_PATH = R5_ROOT / 'training_history.csv'
FUSION_DIAGNOSTIC_ROOT = (
    R5_ROOT / 'diagnostics' / 'fusion_attribution'
    / FUSION_DIAGNOSTIC_VERSION
)
FUSION_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)

CASE_METRICS_PATH = FUSION_DIAGNOSTIC_ROOT / 'case_metrics.csv'
COMPONENT_METRICS_PATH = FUSION_DIAGNOSTIC_ROOT / 'component_metrics.csv'
GATE_METRICS_PATH = FUSION_DIAGNOSTIC_ROOT / 'gate_metrics.csv'
SCENARIO_SUMMARY_PATH = FUSION_DIAGNOSTIC_ROOT / 'scenario_summary.csv'
SIZE_SUMMARY_PATH = FUSION_DIAGNOSTIC_ROOT / 'size_bin_summary.csv'
BRANCH_EFFECT_PATH = FUSION_DIAGNOSTIC_ROOT / 'branch_effect_summary.csv'
INTERACTION_PATH = FUSION_DIAGNOSTIC_ROOT / 'interaction_summary.csv'
PROPOSAL_DISTANCE_PATH = (
    FUSION_DIAGNOSTIC_ROOT / 'proposal_distance_summary.csv'
)
CHECKPOINT_COMPARISON_PATH = (
    FUSION_DIAGNOSTIC_ROOT / 'checkpoint_comparison.csv'
)
CHECKPOINT_SELECTION_AUDIT_PATH = (
    FUSION_DIAGNOSTIC_ROOT / 'checkpoint_selection_audit.csv'
)
FUSION_DIAGNOSTIC_GATE_PATH = (
    FUSION_DIAGNOSTIC_ROOT / 'fusion_attribution_gate.json'
)


FUSION_SCENARIOS = {
    'full': frozenset(),
    'no_global': frozenset({'global'}),
    'no_arterial': frozenset({'arterial'}),
    'no_local_graph': frozenset({'local_graph'}),
    'no_vascular': frozenset({'vascular'}),
    'no_clinical': frozenset({'clinical'}),
    'no_occlusion': frozenset({'occlusion'}),
    'no_atlas_all': frozenset({'atlas_all'}),
    'no_global_no_arterial': frozenset({'global', 'arterial'}),
    'no_global_no_vascular': frozenset({'global', 'vascular'}),
}

CHECKPOINT_LABELS = {
    'best': R5_BEST_CHECKPOINT_PATH,
    'last': R5_LAST_CHECKPOINT_PATH,
}

GATE_OUTPUTS = {
    'proposal_fusion': 'proposal_fusion_gate',
    'context_fusion': 'context_fusion_gate',
    'vascular_quarter': 'vascular_quarter_gate',
    'vascular_eighth': 'vascular_eighth_gate',
    'vascular_bottleneck': 'vascular_bottleneck_gate',
}


def fusion_require_runtime_contract():
    missing = [
        name for name in FUSION_BOOTSTRAP_REQUIRED if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            'Run the complete r5 training block once in this runtime before '
            'the fusion diagnostic. Missing symbols: ' + ', '.join(missing)
        )
    checks = {
        'active_pilot_is_r5': PILOT_VERSION == EXPECTED_R5_PILOT_VERSION,
        'active_code_revision_matches': (
            PILOT_CODE_REVISION == EXPECTED_R5_CODE_REVISION
        ),
        'heldout_has_20_subjects': len(fold0_heldout_df) == 20,
        'heldout_subjects_unique': (
            fold0_heldout_df['subject_id'].astype(str).nunique() == 20
        ),
        'heldout_subjects_are_development_train_only': set(
            fold0_heldout_df['dev_split'].astype(str)
        ) == {'train'},
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'Fusion diagnostic runtime contract failed: ' + ', '.join(failed)
        )
    return checks


def fusion_checkpoint_contract(path, expected_label):
    
    
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    checks = {
        'pilot_version': (
            checkpoint.get('pilot_version') == EXPECTED_R5_PILOT_VERSION
        ),
        'code_revision': (
            checkpoint.get('code_revision') == EXPECTED_R5_CODE_REVISION
        ),
        'parameter_count': (
            int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'fresh_start_lineage': (
            checkpoint.get('fresh_start_from_initialization') is True
        ),
        'parent_migration_disabled': (
            checkpoint.get('parent_checkpoint_migration_allowed') is False
        ),
        'epoch_positive': int(checkpoint.get('epoch', 0)) >= 1,
        'ema_state_present': 'ema_model_state' in checkpoint,
    }
    if expected_label == 'best':
        checks['best_checkpoint_epoch_matches'] = (
            int(checkpoint['epoch']) == int(checkpoint['best_epoch'])
        )
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'Unsafe {expected_label} checkpoint: ' + ', '.join(failed)
        )
    return checkpoint, checks


def fusion_scenario_flags(disabled):
    disabled = set(disabled)
    atlas_all = 'atlas_all' in disabled
    return {
        'cache_force_atlas': atlas_all,
        'cache_force_arterial': atlas_all or 'arterial' in disabled,
        'cache_force_clinical': 'clinical' in disabled,
        'cache_force_occlusion': 'occlusion' in disabled,
        'local_force_global': 'global' in disabled,
        'local_force_atlas': atlas_all,
        'local_zero_graph': atlas_all or 'local_graph' in disabled,
        'local_force_vascular': 'vascular' in disabled,
        'local_force_clinical': 'clinical' in disabled,
        'local_force_occlusion': 'occlusion' in disabled,
    }


def fusion_local_from_cache(model, batch, cache, flags):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(flags['local_force_global'])
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )

    local_inputs = {
        'region_map': batch['region_map'],
        'territory_features': batch['territory_features'],
        'territory_feature_valid': batch['territory_feature_valid'],
        'territory_valid': batch['territory_valid'],
        'territory_reliability': batch['territory_reliability'],
        'vessel_features': batch['vessel_features'],
        'vessel_feature_valid': batch['vessel_feature_valid'],
        'vessel_valid': batch['vessel_valid'],
        'case_features': batch['case_features'],
        'atlas_enabled': batch['atlas_enabled'],
    }
    if flags['local_zero_graph']:
        for name in [
            'region_map', 'territory_features', 'territory_feature_valid',
            'territory_valid', 'territory_reliability', 'vessel_features',
            'vessel_feature_valid', 'vessel_valid', 'case_features',
            'atlas_enabled',
        ]:
            local_inputs[name] = torch.zeros_like(local_inputs[name])

    local = model.local_model(
        image=conditioned_image,
        **local_inputs,
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=flags['local_force_atlas'],
        force_vascular_dropout=flags['local_force_vascular'],
        force_occlusion_dropout=flags['local_force_occlusion'],
        force_clinical_dropout=flags['local_force_clinical'],
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


@torch.no_grad()
def fusion_infer_loaded_case(model, case, device, disabled):
    flags = fusion_scenario_flags(disabled)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch,
            force_atlas_dropout=flags['cache_force_atlas'],
            force_clinical_dropout=flags['cache_force_clinical'],
            force_occlusion_dropout=flags['cache_force_occlusion'],
            force_arterial_dropout=flags['cache_force_arterial'],
        )

    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    gate_values = defaultdict(list)
    windows = 0
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = fusion_local_from_cache(
                        model, batch, cache, flags
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
                for gate_name, output_name in GATE_OUTPUTS.items():
                    values = output[output_name].detach().float().cpu().numpy()
                    gate_values[gate_name].append(values.reshape(-1))
                windows += 1

    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete fusion diagnostic coverage."
        )
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    arterial_canvas = F.interpolate(
        cache['arterial_map'][:, None].float(),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    return {
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal_canvas[native_slices].astype(np.float16),
        'arterial_risk': arterial_canvas[native_slices].astype(np.float16),
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
        'gate_values': {
            name: np.concatenate(values).astype(np.float32)
            for name, values in gate_values.items()
        },
        'windows': windows,
    }


def fusion_case_row(
    checkpoint_label, checkpoint_epoch, scenario, case, result,
    local_threshold, proposal_threshold,
):
    target = case['target']
    prediction = result['probability'] >= local_threshold
    proposal_prediction = (
        result['proposal_probability'] >= proposal_threshold
    )
    intersection = int((prediction & target).sum())
    predicted_voxels = int(prediction.sum())
    target_voxels = int(target.sum())
    proposal_intersection = int((proposal_prediction & target).sum())
    proposal_voxels = int(proposal_prediction.sum())
    voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
    hard_eligible = as_bool(
        case['row']['trial8_hard_metric_eligible']
    )
    dice = (
        2.0 * intersection / max(predicted_voxels + target_voxels, 1)
        if predicted_voxels + target_voxels > 0 else 1.0
    )
    proposal_dice = (
        2.0 * proposal_intersection
        / max(proposal_voxels + target_voxels, 1)
        if proposal_voxels + target_voxels > 0 else 1.0
    )
    lesion_tp, lesion_fp, lesion_fn = pilot_lesion_counts(
        prediction, target
    ) if hard_eligible else (0, 0, 0)
    return {
        'checkpoint': checkpoint_label,
        'checkpoint_epoch': int(checkpoint_epoch),
        'scenario': scenario,
        'subject_id': case['subject_id'],
        'dev_split': str(case['row']['dev_split']),
        'lesion_volume_bin': str(
            case['row']['trial8_effective_lesion_volume_bin']
        ),
        'hard_metric_eligible': hard_eligible,
        'atlas_enabled': as_bool(case['row']['trial8_atlas_enabled']),
        'local_threshold': float(local_threshold),
        'proposal_threshold': float(proposal_threshold),
        'target_voxels': target_voxels,
        'predicted_voxels': predicted_voxels,
        'proposal_voxels': proposal_voxels,
        'intersection_voxels': intersection,
        'proposal_intersection_voxels': proposal_intersection,
        'dice': float(dice),
        'proposal_dice': float(proposal_dice),
        'voxel_precision': intersection / max(predicted_voxels, 1),
        'voxel_recall': intersection / max(target_voxels, 1),
        'predicted_false_positive_volume_ml': int(
            (prediction & ~target).sum()
        ) * voxel_volume_ml,
        'target_volume_ml': target_voxels * voxel_volume_ml,
        'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
        'predicted_volume_ml_from_router': result['predicted_volume_ml'],
        'presence_probability': result['presence_probability'],
        'presence_target': float(target_voxels > 0),
        'lesion_tp': int(lesion_tp),
        'lesion_fp': int(lesion_fp),
        'lesion_fn': int(lesion_fn),
        'windows': int(result['windows']),
    }


def fusion_component_rows(
    checkpoint_label, checkpoint_epoch, scenario, case, result,
    local_threshold, proposal_threshold,
):
    target = case['target']
    prediction = result['probability'] >= local_threshold
    proposal_prediction = (
        result['proposal_probability'] >= proposal_threshold
    )
    final_iou = {
        int(item['component_id']): item
        for item in pilot_target_component_iou_records(
            prediction, target, case['spacing']
        )
    }
    proposal_iou = {
        int(item['component_id']): item
        for item in pilot_target_component_iou_records(
            proposal_prediction, target, case['spacing']
        )
    }
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    component_objects = ndimage.find_objects(labels)
    proposal_distance = None
    if scenario == 'full':
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
    rows = []
    voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
    for component_id, component_slice in enumerate(
        component_objects, start=1
    ):
        if component_slice is None:
            continue
        component_local = labels[component_slice] == component_id
        voxels = int(component_local.sum())
        volume_ml = voxels * voxel_volume_ml
        final_max_probability = float(
            result['probability'][component_slice][component_local].max(
                initial=0.0
            )
        )
        proposal_max_probability = float(
            result['proposal_probability'][component_slice][
                component_local
            ].max(initial=0.0)
        )
        arterial_max = float(
            result['arterial_risk'][component_slice][component_local].max(
                initial=0.0
            )
        )
        distance_mm = float('nan')
        if scenario == 'full':
            if proposal_distance is not None:
                distance_mm = float(
                    proposal_distance[component_slice][component_local].min(
                        initial=np.inf
                    )
                )
        final_max_iou = float(
            final_iou[component_id]['maximum_component_iou']
        )
        proposal_max_iou = float(
            proposal_iou[component_id]['maximum_component_iou']
        )
        rows.append({
            'checkpoint': checkpoint_label,
            'checkpoint_epoch': int(checkpoint_epoch),
            'scenario': scenario,
            'subject_id': case['subject_id'],
            'lesion_volume_bin': str(
                case['row']['trial8_effective_lesion_volume_bin']
            ),
            'atlas_enabled': as_bool(case['row']['trial8_atlas_enabled']),
            'component_id': int(component_id),
            'component_tier': pilot_component_tier(volume_ml, voxels),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'final_maximum_probability': final_max_probability,
            'proposal_maximum_probability': proposal_max_probability,
            'arterial_risk_maximum': arterial_max,
            'final_maximum_iou': final_max_iou,
            'proposal_maximum_iou': proposal_max_iou,
            'final_detected_iou_0p10': final_max_iou >= 0.10,
            'proposal_detected_iou_0p05': proposal_max_iou >= 0.05,
            'proposal_minimum_distance_mm': distance_mm,
            'proposal_within_2mm': bool(
                np.isfinite(distance_mm) and distance_mm <= 2.0
            ),
            'proposal_within_4mm': bool(
                np.isfinite(distance_mm) and distance_mm <= 4.0
            ),
            'proposal_within_8mm': bool(
                np.isfinite(distance_mm) and distance_mm <= 8.0
            ),
            'representation_failure_below_0p10': (
                final_max_probability < 0.10
            ),
        })
    if len(rows) != int(count):
        raise RuntimeError(
            f"{case['subject_id']}: component count changed during audit."
        )
    return rows


def fusion_gate_rows(
    checkpoint_label, checkpoint_epoch, scenario, case, result,
):
    rows = []
    for gate_name, values in result['gate_values'].items():
        absolute = np.abs(values.astype(np.float64))
        rows.append({
            'checkpoint': checkpoint_label,
            'checkpoint_epoch': int(checkpoint_epoch),
            'scenario': scenario,
            'subject_id': case['subject_id'],
            'lesion_volume_bin': str(
                case['row']['trial8_effective_lesion_volume_bin']
            ),
            'gate': gate_name,
            'gate_values': int(values.size),
            'mean_gate': float(values.mean()),
            'mean_absolute_gate': float(absolute.mean()),
            'maximum_absolute_gate': float(absolute.max(initial=0.0)),
            'fraction_absolute_at_least_0p09': float(
                np.mean(absolute >= 0.09)
            ),
        })
    return rows


def fusion_aggregate_scenarios(case_df, component_df):
    rows = []
    for (checkpoint, scenario), frame in case_df.groupby(
        ['checkpoint', 'scenario'], sort=True
    ):
        positive = frame[
            frame['hard_metric_eligible'] & (frame['target_voxels'] > 0)
        ]
        hard = frame[frame['hard_metric_eligible']]
        components = component_df[
            (component_df['checkpoint'] == checkpoint)
            & (component_df['scenario'] == scenario)
        ]
        precision = hard['lesion_tp'].sum() / max(
            hard['lesion_tp'].sum() + hard['lesion_fp'].sum(), 1
        )
        recall = hard['lesion_tp'].sum() / max(
            hard['lesion_tp'].sum() + hard['lesion_fn'].sum(), 1
        )
        lesion_f1 = (
            2.0 * precision * recall / max(precision + recall, 1e-12)
        )
        rows.append({
            'checkpoint': checkpoint,
            'scenario': scenario,
            'subjects': len(frame),
            'positive_subjects': len(positive),
            'mean_dice': float(positive['dice'].mean()),
            'median_dice': float(positive['dice'].median()),
            'global_dice': float(
                2.0 * positive['intersection_voxels'].sum()
                / max(
                    positive['predicted_voxels'].sum()
                    + positive['target_voxels'].sum(), 1
                )
            ),
            'mean_proposal_dice': float(positive['proposal_dice'].mean()),
            'global_proposal_dice': float(
                2.0 * positive['proposal_intersection_voxels'].sum()
                / max(
                    positive['proposal_voxels'].sum()
                    + positive['target_voxels'].sum(), 1
                )
            ),
            'lesion_precision': float(precision),
            'lesion_recall': float(recall),
            'lesion_f1': float(lesion_f1),
            'component_recall': float(
                components['final_detected_iou_0p10'].mean()
            ) if len(components) else 0.0,
            'proposal_component_recall': float(
                components['proposal_detected_iou_0p05'].mean()
            ) if len(components) else 0.0,
            'mean_false_positive_volume_ml': float(
                hard['predicted_false_positive_volume_ml'].mean()
            ),
            'presence_brier_score': float(np.mean(
                (
                    hard['presence_probability']
                    - hard['presence_target']
                ) ** 2
            )),
            'router_volume_mae_ml': float(np.mean(np.abs(
                positive['predicted_volume_ml_from_router']
                - positive['target_volume_ml']
            ))),
        })
    return pd.DataFrame(rows)


def fusion_size_summary(case_df, component_df):
    rows = []
    positive = case_df[
        case_df['hard_metric_eligible'] & (case_df['target_voxels'] > 0)
    ]
    for keys, frame in positive.groupby(
        ['checkpoint', 'scenario', 'lesion_volume_bin'], sort=True
    ):
        checkpoint, scenario, lesion_bin = keys
        components = component_df[
            (component_df['checkpoint'] == checkpoint)
            & (component_df['scenario'] == scenario)
            & (component_df['lesion_volume_bin'] == lesion_bin)
        ]
        rows.append({
            'checkpoint': checkpoint,
            'scenario': scenario,
            'lesion_volume_bin': lesion_bin,
            'subjects': len(frame),
            'mean_dice': float(frame['dice'].mean()),
            'median_dice': float(frame['dice'].median()),
            'component_recall': float(
                components['final_detected_iou_0p10'].mean()
            ) if len(components) else 0.0,
            'proposal_component_recall': float(
                components['proposal_detected_iou_0p05'].mean()
            ) if len(components) else 0.0,
        })
    return pd.DataFrame(rows)


def fusion_branch_effects(case_df):
    positive = case_df[
        case_df['hard_metric_eligible'] & (case_df['target_voxels'] > 0)
    ].copy()
    full = positive[positive['scenario'] == 'full'][
        ['checkpoint', 'subject_id', 'dice', 'predicted_false_positive_volume_ml']
    ].rename(columns={
        'dice': 'full_dice',
        'predicted_false_positive_volume_ml': 'full_fp_volume_ml',
    })
    merged = positive.merge(full, on=['checkpoint', 'subject_id'], how='left')
    merged['dice_contribution_when_present'] = (
        merged['full_dice'] - merged['dice']
    )
    merged['fp_volume_change_when_removed_ml'] = (
        merged['predicted_false_positive_volume_ml']
        - merged['full_fp_volume_ml']
    )
    rows = []
    for keys, frame in merged.groupby(
        ['checkpoint', 'scenario', 'lesion_volume_bin'], sort=True
    ):
        rows.append({
            'checkpoint': keys[0], 'scenario': keys[1],
            'lesion_volume_bin': keys[2], 'subjects': len(frame),
            'mean_dice_contribution_when_present': float(
                frame['dice_contribution_when_present'].mean()
            ),
            'median_dice_contribution_when_present': float(
                frame['dice_contribution_when_present'].median()
            ),
            'mean_fp_volume_change_when_removed_ml': float(
                frame['fp_volume_change_when_removed_ml'].mean()
            ),
        })
    overall = merged.groupby(['checkpoint', 'scenario']).agg(
        subjects=('subject_id', 'size'),
        mean_dice_contribution_when_present=(
            'dice_contribution_when_present', 'mean'
        ),
        median_dice_contribution_when_present=(
            'dice_contribution_when_present', 'median'
        ),
        mean_fp_volume_change_when_removed_ml=(
            'fp_volume_change_when_removed_ml', 'mean'
        ),
    ).reset_index()
    overall['lesion_volume_bin'] = 'all_positive'
    return pd.concat([overall, pd.DataFrame(rows)], ignore_index=True)


def fusion_interactions(case_df):
    positive = case_df[
        case_df['hard_metric_eligible'] & (case_df['target_voxels'] > 0)
    ]
    pivot = positive.pivot_table(
        index=['checkpoint', 'subject_id', 'lesion_volume_bin'],
        columns='scenario', values='dice', aggfunc='first'
    ).reset_index()
    definitions = {
        'global_x_arterial': (
            'no_global', 'no_arterial', 'no_global_no_arterial'
        ),
        'global_x_vascular': (
            'no_global', 'no_vascular', 'no_global_no_vascular'
        ),
    }
    rows = []
    for name, (no_a, no_b, no_both) in definitions.items():
        values = (
            pivot['full'] - pivot[no_a] - pivot[no_b] + pivot[no_both]
        )
        frame = pivot.assign(interaction=values)
        for keys, group in frame.groupby(
            ['checkpoint', 'lesion_volume_bin'], sort=True
        ):
            rows.append({
                'checkpoint': keys[0], 'interaction': name,
                'lesion_volume_bin': keys[1], 'subjects': len(group),
                'mean_dice_interaction': float(group['interaction'].mean()),
                'median_dice_interaction': float(
                    group['interaction'].median()
                ),
            })
        for checkpoint, group in frame.groupby('checkpoint', sort=True):
            rows.append({
                'checkpoint': checkpoint, 'interaction': name,
                'lesion_volume_bin': 'all_positive',
                'subjects': len(group),
                'mean_dice_interaction': float(group['interaction'].mean()),
                'median_dice_interaction': float(
                    group['interaction'].median()
                ),
            })
    return pd.DataFrame(rows)


def fusion_proposal_distance_summary(component_df):
    frame = component_df[component_df['scenario'] == 'full'].copy()
    rows = []
    for keys, group in frame.groupby(
        ['checkpoint', 'component_tier'], sort=True
    ):
        finite = group[np.isfinite(group['proposal_minimum_distance_mm'])]
        rows.append({
            'checkpoint': keys[0], 'component_tier': keys[1],
            'components': len(group),
            'proposal_iou_recall_0p05': float(
                group['proposal_detected_iou_0p05'].mean()
            ),
            'proposal_within_2mm_fraction': float(
                group['proposal_within_2mm'].mean()
            ),
            'proposal_within_4mm_fraction': float(
                group['proposal_within_4mm'].mean()
            ),
            'proposal_within_8mm_fraction': float(
                group['proposal_within_8mm'].mean()
            ),
            'median_finite_proposal_distance_mm': float(
                finite['proposal_minimum_distance_mm'].median()
            ) if len(finite) else float('nan'),
            'mean_proposal_maximum_probability': float(
                group['proposal_maximum_probability'].mean()
            ),
            'mean_final_maximum_probability': float(
                group['final_maximum_probability'].mean()
            ),
            'representation_failure_fraction': float(
                group['representation_failure_below_0p10'].mean()
            ),
            'mean_arterial_risk_maximum': float(
                group['arterial_risk_maximum'].mean()
            ),
        })
    return pd.DataFrame(rows)


def fusion_checkpoint_comparison(scenario_summary):
    full = scenario_summary[
        scenario_summary['scenario'] == 'full'
    ].set_index('checkpoint')
    if not {'best', 'last'}.issubset(full.index):
        raise RuntimeError('Best/last full-model summaries are incomplete.')
    metrics = [
        'mean_dice', 'global_dice', 'lesion_f1', 'component_recall',
        'proposal_component_recall', 'global_proposal_dice',
    ]
    return pd.DataFrame([{
        'metric': metric,
        'best': float(full.loc['best', metric]),
        'last': float(full.loc['last', metric]),
        'last_minus_best': float(
            full.loc['last', metric] - full.loc['best', metric]
        ),
    } for metric in metrics])


def fusion_checkpoint_selection_audit(history, r5_result):
    required = {
        'epoch', 'pilot_checkpoint_score', 'validation_mean_dice',
        'validation_global_dice', 'best_epoch', 'improved',
    }
    missing = sorted(required - set(history.columns))
    if missing:
        raise RuntimeError(
            'R5 history lacks checkpoint-selection fields: '
            + ', '.join(missing)
        )
    strict_row = history.loc[
        history['pilot_checkpoint_score'].astype(float).idxmax()
    ]
    saved_best_epoch = int(r5_result['best_epoch'])
    saved_row = history[
        history['epoch'].astype(int) == saved_best_epoch
    ]
    if len(saved_row) != 1:
        raise RuntimeError('R5 saved-best epoch is absent or duplicated.')
    saved_row = saved_row.iloc[0]
    return pd.DataFrame([{
        'saved_best_epoch': saved_best_epoch,
        'saved_best_score': float(saved_row['pilot_checkpoint_score']),
        'saved_best_mean_dice': float(saved_row['validation_mean_dice']),
        'saved_best_global_dice': float(saved_row['validation_global_dice']),
        'strict_numeric_best_epoch': int(strict_row['epoch']),
        'strict_numeric_best_score': float(
            strict_row['pilot_checkpoint_score']
        ),
        'strict_numeric_best_mean_dice': float(
            strict_row['validation_mean_dice']
        ),
        'strict_numeric_best_global_dice': float(
            strict_row['validation_global_dice']
        ),
        'strict_score_minus_saved_score': float(
            strict_row['pilot_checkpoint_score']
            - saved_row['pilot_checkpoint_score']
        ),
        'minimum_delta_coupled_to_checkpoint_selection': (
            int(strict_row['epoch']) != saved_best_epoch
        ),
        'recommended_policy': (
            'save every strict numeric best; use minimum delta only to reset '
            'early-stopping patience'
        ),
    }])


def fusion_evidence_summary(
    scenario_summary, branch_effect_df, interaction_df,
    distance_df, checkpoint_df, gate_df, selection_df,
):
    best_full = scenario_summary[
        (scenario_summary['checkpoint'] == 'best')
        & (scenario_summary['scenario'] == 'full')
    ].iloc[0]
    best_distance = distance_df[distance_df['checkpoint'] == 'best']
    tiny_tiers = best_distance[
        best_distance['component_tier'].isin([
            'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
        ])
    ]
    weighted_components = max(int(tiny_tiers['components'].sum()), 1)
    near_8 = float(
        (
            tiny_tiers['proposal_within_8mm_fraction']
            * tiny_tiers['components']
        ).sum() / weighted_components
    )
    representation_failure = float(
        (
            tiny_tiers['representation_failure_fraction']
            * tiny_tiers['components']
        ).sum() / weighted_components
    )
    overall_effect = branch_effect_df[
        (branch_effect_df['checkpoint'] == 'best')
        & (branch_effect_df['lesion_volume_bin'] == 'all_positive')
        & (branch_effect_df['scenario'] != 'full')
    ].sort_values('mean_dice_contribution_when_present', ascending=False)
    strongest = (
        str(overall_effect.iloc[0]['scenario']) if len(overall_effect)
        else 'none'
    )
    best_to_last_dice = float(
        checkpoint_df.loc[
            checkpoint_df['metric'] == 'mean_dice', 'last_minus_best'
        ].iloc[0]
    )
    proposal_recall = float(best_full['proposal_component_recall'])
    final_recall = float(best_full['component_recall'])
    if proposal_recall < 0.25 and near_8 >= 0.50:
        primary = 'proposal_resolution_or_calibration_bottleneck'
    elif proposal_recall < 0.25 and near_8 < 0.50:
        primary = 'proposal_representation_or_supervision_bottleneck'
    elif proposal_recall - final_recall >= 0.15:
        primary = 'proposal_to_local_fusion_bottleneck'
    else:
        primary = 'mixed_bottleneck'
    return {
        'primary_evidence_class': primary,
        'best_mean_dice_fixed_threshold': float(best_full['mean_dice']),
        'best_global_dice_fixed_threshold': float(best_full['global_dice']),
        'best_proposal_component_recall': proposal_recall,
        'best_final_component_recall': final_recall,
        'small_component_proposal_within_8mm_fraction': near_8,
        'small_component_representation_failure_fraction': (
            representation_failure
        ),
        'strongest_positive_branch_by_leave_one_out': strongest,
        'last_minus_best_mean_dice': best_to_last_dice,
        'overfit_confirmed': best_to_last_dice < -0.002,
        'gate_saturation_present': bool(
            (gate_df['fraction_absolute_at_least_0p09'] > 0.25).any()
        ),
        'checkpoint_selection_should_be_decoupled_from_patience': bool(
            selection_df.iloc[0][
                'minimum_delta_coupled_to_checkpoint_selection'
            ]
        ),
        'interpretation_requires_manual_review': True,
    }


def run_fusion_attribution_diagnostic():
    runtime_checks = fusion_require_runtime_contract()
    required_paths = [
        R5_BEST_CHECKPOINT_PATH, R5_LAST_CHECKPOINT_PATH,
        R5_RESULT_GATE_PATH, R5_HISTORY_PATH,
    ]
    missing = [str(path) for path in required_paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'R5 fusion diagnostic prerequisites are missing:\n'
            + '\n'.join(missing)
        )
    source_hashes_before = {
        str(path): file_sha256(path) for path in required_paths
    }
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_history = pd.read_csv(R5_HISTORY_PATH)
    result_checks = {
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r5_version_matches': (
            r5_result.get('pilot_version') == EXPECTED_R5_PILOT_VERSION
        ),
        'r5_code_revision_matches': (
            r5_result.get('code_revision') == EXPECTED_R5_CODE_REVISION
        ),
        'r5_fresh_start_lineage': (
            r5_result.get('fresh_start_from_initialization') is True
        ),
        'r5_parent_migration_disabled': (
            r5_result.get('parent_checkpoint_migration_allowed') is False
        ),
        'r5_fivefold_training_locked': (
            r5_result.get('full_fivefold_training_authorized') is False
        ),
    }
    if not all(result_checks.values()):
        failed = [name for name, passed in result_checks.items() if not passed]
        raise RuntimeError('R5 result gate failed: ' + ', '.join(failed))

    if FUSION_DIAGNOSTIC_GATE_PATH.exists():
        completed = load_json(FUSION_DIAGNOSTIC_GATE_PATH)
        if (
            completed.get('diagnostic_complete') is True
            and completed.get('code_revision')
            == FUSION_DIAGNOSTIC_CODE_REVISION
            and completed.get('source_checkpoint_sha256')
            == source_hashes_before
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R5 FUSION ATTRIBUTION ALREADY COMPLETE')
            print('=' * 80)
            print(f'Diagnostic gate: {FUSION_DIAGNOSTIC_GATE_PATH}')
            print('No diagnostic or checkpoint artifact was rewritten.')
            return

    if not torch.cuda.is_available():
        raise RuntimeError('Fusion attribution diagnostic requires a CUDA GPU.')
    device = torch.device('cuda')
    case_rows = []
    component_rows = []
    gate_rows = []
    checkpoint_contracts = {}
    best_threshold = None
    proposal_threshold = None
    start_time = time.time()

    for checkpoint_label, checkpoint_path in CHECKPOINT_LABELS.items():
        checkpoint, checkpoint_checks = fusion_checkpoint_contract(
            checkpoint_path, checkpoint_label
        )
        checkpoint_contracts[checkpoint_label] = checkpoint_checks
        if checkpoint_label == 'best':
            best_threshold = float(checkpoint['best_threshold'])
            proposal_threshold = float(
                checkpoint['best_proposal_threshold']
            )
        model = VascuFateNetV22(pilot_schema).to(device)
        model.load_state_dict(checkpoint['ema_model_state'])
        model.eval()
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
            raise RuntimeError('Fusion diagnostic model parameter count changed.')
        if any(parameter.requires_grad for parameter in model.parameters()):
            raise RuntimeError('Diagnostic model unexpectedly remains trainable.')

        checkpoint_epoch = int(checkpoint['epoch'])
        for case_index, row in enumerate(heldout_rows, start=1):
            case = pilot_load_validation_case(row)
            for scenario, disabled in FUSION_SCENARIOS.items():
                result = fusion_infer_loaded_case(
                    model, case, device, disabled
                )
                case_rows.append(fusion_case_row(
                    checkpoint_label, checkpoint_epoch, scenario, case,
                    result, best_threshold, proposal_threshold,
                ))
                component_rows.extend(fusion_component_rows(
                    checkpoint_label, checkpoint_epoch, scenario, case,
                    result, best_threshold, proposal_threshold,
                ))
                gate_rows.extend(fusion_gate_rows(
                    checkpoint_label, checkpoint_epoch, scenario, case,
                    result,
                ))
                del result
            print(
                f'{checkpoint_label.capitalize()} checkpoint '
                f'{case_index:02d}/{len(heldout_rows)}: '
                f"{case['subject_id']}, scenarios={len(FUSION_SCENARIOS)}"
            )
            del case
            gc.collect()
            torch.cuda.empty_cache()
        del model, checkpoint
        gc.collect()
        torch.cuda.empty_cache()

    case_df = pd.DataFrame(case_rows)
    component_df = pd.DataFrame(component_rows)
    gate_df = pd.DataFrame(gate_rows)
    scenario_summary = fusion_aggregate_scenarios(case_df, component_df)
    size_summary = fusion_size_summary(case_df, component_df)
    branch_effect_df = fusion_branch_effects(case_df)
    interaction_df = fusion_interactions(case_df)
    distance_df = fusion_proposal_distance_summary(component_df)
    checkpoint_df = fusion_checkpoint_comparison(scenario_summary)
    selection_df = fusion_checkpoint_selection_audit(
        r5_history, r5_result
    )

    best_full = scenario_summary[
        (scenario_summary['checkpoint'] == 'best')
        & (scenario_summary['scenario'] == 'full')
    ].iloc[0]
    expected_best = r5_result['best_validation_summary']
    reproduction_errors = {
        'mean_dice': abs(
            float(best_full['mean_dice'])
            - float(expected_best['positive_mean_dice'])
        ),
        'global_dice': abs(
            float(best_full['global_dice'])
            - float(expected_best['positive_global_dice'])
        ),
        'component_recall': abs(
            float(best_full['component_recall'])
            - float(expected_best['component_recall'])
        ),
        'proposal_component_recall': abs(
            float(best_full['proposal_component_recall'])
            - float(expected_best['proposal_component_recall'])
        ),
    }

    source_hashes_after = {
        str(path): file_sha256(path) for path in required_paths
    }
    evidence = fusion_evidence_summary(
        scenario_summary, branch_effect_df, interaction_df,
        distance_df, checkpoint_df, gate_df, selection_df,
    )
    checks = {
        **runtime_checks,
        **{f'result_{name}': value for name, value in result_checks.items()},
        'best_checkpoint_contract_passed': all(
            checkpoint_contracts['best'].values()
        ),
        'last_checkpoint_contract_passed': all(
            checkpoint_contracts['last'].values()
        ),
        'all_20_subjects_audited': (
            case_df['subject_id'].nunique() == 20
        ),
        'both_checkpoints_audited': set(case_df['checkpoint'])
        == {'best', 'last'},
        'all_scenarios_audited': set(case_df['scenario'])
        == set(FUSION_SCENARIOS),
        'case_matrix_complete': len(case_df)
        == 2 * 20 * len(FUSION_SCENARIOS),
        'all_case_probabilistic_metrics_finite': bool(np.isfinite(
            case_df[[
                'dice', 'proposal_dice', 'voxel_precision', 'voxel_recall',
                'presence_probability', 'predicted_volume_ml_from_router',
            ]].to_numpy(dtype=np.float64)
        ).all()),
        'all_gate_metrics_finite': bool(np.isfinite(
            gate_df[[
                'mean_gate', 'mean_absolute_gate',
                'maximum_absolute_gate',
                'fraction_absolute_at_least_0p09',
            ]].to_numpy(dtype=np.float64)
        ).all()),
        'fixed_threshold_matches_r5_best': (
            abs(best_threshold - float(r5_result['best_threshold'])) < 1e-12
            and abs(
                proposal_threshold
                - float(r5_result['best_proposal_threshold'])
            ) < 1e-12
        ),
        'best_checkpoint_reproduced_within_float16_tolerance': max(
            reproduction_errors.values()
        ) <= 5e-4,
        'source_checkpoint_hashes_unchanged': (
            source_hashes_before == source_hashes_after
        ),
        'history_has_one_row_per_completed_epoch': (
            r5_history['epoch'].astype(int).nunique() == len(r5_history)
        ),
        'target_arrays_read_from_train_split_only': set(
            case_df['dev_split']
        ) == {'train'},
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'outcome_workbooks_not_read': True,
        'optimizer_was_not_constructed': True,
        'backward_was_not_called': True,
        'optimizer_steps_are_zero': True,
        'study_trainable_parameters_are_zero': True,
        'full_fivefold_training_remains_locked': True,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R5 FUSION ATTRIBUTION FAILED. Failed checks: '
            + ', '.join(failed)
        )

    atomic_write_csv(case_df, CASE_METRICS_PATH)
    atomic_write_csv(component_df, COMPONENT_METRICS_PATH)
    atomic_write_csv(gate_df, GATE_METRICS_PATH)
    atomic_write_csv(scenario_summary, SCENARIO_SUMMARY_PATH)
    atomic_write_csv(size_summary, SIZE_SUMMARY_PATH)
    atomic_write_csv(branch_effect_df, BRANCH_EFFECT_PATH)
    atomic_write_csv(interaction_df, INTERACTION_PATH)
    atomic_write_csv(distance_df, PROPOSAL_DISTANCE_PATH)
    atomic_write_csv(checkpoint_df, CHECKPOINT_COMPARISON_PATH)
    atomic_write_csv(selection_df, CHECKPOINT_SELECTION_AUDIT_PATH)

    diagnostic_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': FUSION_DIAGNOSTIC_VERSION,
        'code_revision': FUSION_DIAGNOSTIC_CODE_REVISION,
        'diagnostic_complete': True,
        'source_checkpoint_sha256': source_hashes_before,
        'best_epoch': int(r5_result['best_epoch']),
        'last_epoch': int(
            case_df.loc[
                case_df['checkpoint'] == 'last', 'checkpoint_epoch'
            ].iloc[0]
        ),
        'local_threshold_fixed_from_best': best_threshold,
        'proposal_threshold_fixed_from_best': proposal_threshold,
        'subjects': 20,
        'scenarios': list(FUSION_SCENARIOS),
        'reproduction_errors': reproduction_errors,
        'evidence_summary': evidence,
        'checks': checks,
        'case_metrics_path': str(CASE_METRICS_PATH),
        'component_metrics_path': str(COMPONENT_METRICS_PATH),
        'gate_metrics_path': str(GATE_METRICS_PATH),
        'scenario_summary_path': str(SCENARIO_SUMMARY_PATH),
        'size_summary_path': str(SIZE_SUMMARY_PATH),
        'branch_effect_path': str(BRANCH_EFFECT_PATH),
        'interaction_path': str(INTERACTION_PATH),
        'proposal_distance_path': str(PROPOSAL_DISTANCE_PATH),
        'checkpoint_comparison_path': str(CHECKPOINT_COMPARISON_PATH),
        'checkpoint_selection_audit_path': str(
            CHECKPOINT_SELECTION_AUDIT_PATH
        ),
        'elapsed_minutes': (time.time() - start_time) / 60.0,
        'target_arrays_read_splits': ['train'],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'study_trainable_parameters': 0,
        'training_performed': False,
        'full_fivefold_training_authorized': False,
        'manual_review_required': True,
    }
    atomic_write_json(diagnostic_gate, FUSION_DIAGNOSTIC_GATE_PATH)

    print('\nScenario summary:')
    display(scenario_summary)
    print('\nLeave-one-out branch effects:')
    display(branch_effect_df[
        branch_effect_df['lesion_volume_bin'] == 'all_positive'
    ].sort_values([
        'checkpoint', 'mean_dice_contribution_when_present'
    ], ascending=[True, False]))
    print('\nPairwise fusion interactions:')
    display(interaction_df[
        interaction_df['lesion_volume_bin'] == 'all_positive'
    ])
    print('\nProposal distance by component tier:')
    display(distance_df)
    print('\nBest-versus-last comparison:')
    display(checkpoint_df)
    print('\nCheckpoint-selection control audit:')
    display(selection_df)

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R5 FUSION ATTRIBUTION DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Subjects:                           20')
    print(f'Checkpoints:                        best and last')
    print(f'Scenarios per checkpoint:           {len(FUSION_SCENARIOS)}')
    print(f'Primary evidence class:             '
          f"{evidence['primary_evidence_class']}")
    print(f'Strongest positive branch:          '
          f"{evidence['strongest_positive_branch_by_leave_one_out']}")
    print(f'Best mean Dice, reproduced:         '
          f"{evidence['best_mean_dice_fixed_threshold']:.4f}")
    print(f'Small proposal within 8 mm:         '
          f"{evidence['small_component_proposal_within_8mm_fraction']:.4f}")
    print(f'Small representation failures:      '
          f"{evidence['small_component_representation_failure_fraction']:.4f}")
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Training performed:                  False')
    print('Full five-fold training authorized:  False')
    print(f'Diagnostic gate:                    '
          f'{FUSION_DIAGNOSTIC_GATE_PATH}')
    print('\nNext: use this audit to choose one isolated r6 intervention; '
          'do not launch five-fold training yet.')


run_fusion_attribution_diagnostic()


In [ ]:
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage


R6_PREFLIGHT_VERSION = 'v1_equalized_selected_component_coverage'
R6_PREFLIGHT_CODE_REVISION = 'r2_exact_frozen_support_policy'
EXPECTED_R5_PILOT_VERSION = (
    'v2_2_fold0_global_local_component_diagnostic_r5'
)
EXPECTED_R5_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
EXPECTED_R5_FUSION_DIAGNOSTIC_VERSION = (
    'v1_all20_best_vs_last_fixed_threshold'
)
EXPECTED_R5_FUSION_DIAGNOSTIC_REVISION = (
    'r3_positive_read_only_gate_assertions'
)

R6_COMPONENT_LOSS_COEFFICIENT = 0.25
R6_EXACT_COVERAGE_WEIGHT = 0.80
R6_TOLERANT_COVERAGE_WEIGHT = 0.20
R6_TOLERANCE_RADIUS_MM = 4.0
R6_TOLERANT_TOPK = 4
R6_NATIVE_SPACING_MM = (0.8, 0.8, 2.0)
R6_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip(R6_NATIVE_SPACING_MM, (2, 2, 2))
)
R6_SYNTHETIC_COMPONENT_SIZES = (1, 2, 4, 8, 32, 128)
R6_EXPECTED_ATOMIC_COMPONENTS = 872
R6_EXPECTED_ARCHITECTURE_PARAMETERS = 16_870_679
R6_MANUAL_R5_DIAGNOSTIC_REVIEW_ACCEPTED = True


R6_BOOTSTRAP_REQUIRED = (
    'PROJECT_ROOT', 'R2_VERSION', 'MODEL_VERSION',
    'PILOT_VERSION', 'PILOT_CODE_REVISION', 'PILOT_ROOT',
    'PILOT_RESULT_GATE_PATH', 'EXPECTED_V22_PARAMETERS',
    'GLOBAL_CANVAS_SHAPE', 'GLOBAL_INPUT_SHAPE', 'GLOBAL_DOWNSAMPLE',
    'fold0_fit_df', 'fold0_heldout_df', 'locked_validation_df',
    'sealed_test_df', 'component_df', 'component_records',
    'pilot_mask_records', 'pilot_unpack_mask',
    'pilot_build_epoch_schedule', 'pilot_soft_dice_loss',
    'pilot_focal_bce_loss', 'atomic_write_csv', 'atomic_write_json',
    'file_sha256', 'load_json', 'as_bool',
)
R6_BOOTSTRAP_MISSING = [
    name for name in R6_BOOTSTRAP_REQUIRED if name not in globals()
]
if R6_BOOTSTRAP_MISSING:
    raise RuntimeError(
        'TRIAL8 R6 COMPONENT-PROPOSAL PREFLIGHT RUNTIME BOOTSTRAP REQUIRED. '
        'Run the complete current trial8_v2_2_fold0_pilot_training.py block '
        'once in this Colab runtime. A completed r5 run should print '
        '"TRIAL8 V2.2 FOLD-0 PILOT ALREADY COMPLETE" and return without '
        'retraining or rewriting model checkpoints. Then rerun this r6 '
        'preflight. Do not delete or rename the r5 directory. Missing runtime '
        'symbols: ' + ', '.join(R6_BOOTSTRAP_MISSING)
    )


R5_FUSION_GATE_PATH = (
    PILOT_ROOT / 'diagnostics' / 'fusion_attribution'
    / EXPECTED_R5_FUSION_DIAGNOSTIC_VERSION
    / 'fusion_attribution_gate.json'
)
R6_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r6_component_preserving_proposal'
    / R6_PREFLIGHT_VERSION
)
R6_COMPONENT_AUDIT_PATH = R6_PREFLIGHT_ROOT / 'component_pooling_audit.csv'
R6_TARGET_ACCESS_AUDIT_PATH = R6_PREFLIGHT_ROOT / 'target_access_audit.csv'
R6_SCHEDULE_AUDIT_PATH = R6_PREFLIGHT_ROOT / 'schedule_component_audit.csv'
R6_SYNTHETIC_GRADIENT_PATH = (
    R6_PREFLIGHT_ROOT / 'synthetic_gradient_allocation_audit.csv'
)
R6_REAL_GRADIENT_PATH = (
    R6_PREFLIGHT_ROOT / 'real_component_gradient_audit.csv'
)
R6_TIER_SUMMARY_PATH = R6_PREFLIGHT_ROOT / 'component_tier_summary.csv'
R6_POLICY_PATH = R6_PREFLIGHT_ROOT / 'r6_proposal_loss_policy.json'
R6_PREFLIGHT_GATE_PATH = (
    R6_PREFLIGHT_ROOT / 'r6_component_proposal_preflight_gate.json'
)


def r6_require_runtime_contract():
    missing_paths = [
        str(path) for path in (PILOT_RESULT_GATE_PATH, R5_FUSION_GATE_PATH)
        if not Path(path).exists()
    ]
    if missing_paths:
        raise FileNotFoundError(
            'TRIAL8 R6 PREFLIGHT PREREQUISITES ARE MISSING. Complete the r5 '
            'fold-0 pilot and its fusion-attribution diagnostic first. '
            'Missing: ' + ', '.join(missing_paths)
        )
    r5_result = load_json(PILOT_RESULT_GATE_PATH)
    fusion_gate = load_json(R5_FUSION_GATE_PATH)
    fusion_checks = fusion_gate.get('checks', {})
    evidence = fusion_gate.get('evidence_summary', {})
    checks = {
        'active_runtime_is_r5': PILOT_VERSION == EXPECTED_R5_PILOT_VERSION,
        'active_runtime_revision_matches_r5': (
            PILOT_CODE_REVISION == EXPECTED_R5_CODE_REVISION
        ),
        'r5_result_gate_exists': Path(PILOT_RESULT_GATE_PATH).exists(),
        'r5_pilot_completed': as_bool(r5_result.get('pilot_complete', False)),
        'r5_training_was_fold0_only': not as_bool(
            r5_result.get('full_fivefold_training_authorized', False)
        ),
        'fusion_gate_exists': R5_FUSION_GATE_PATH.exists(),
        'fusion_diagnostic_complete': as_bool(
            fusion_gate.get('diagnostic_complete', False)
        ),
        'fusion_diagnostic_version_matches': (
            fusion_gate.get('diagnostic_version')
            == EXPECTED_R5_FUSION_DIAGNOSTIC_VERSION
        ),
        'fusion_diagnostic_revision_matches': (
            fusion_gate.get('code_revision')
            == EXPECTED_R5_FUSION_DIAGNOSTIC_REVISION
        ),
        'all_fusion_diagnostic_checks_passed': (
            len(fusion_checks) > 0
            and all(bool(value) for value in fusion_checks.values())
        ),
        'diagnosed_bottleneck_matches_r6_intervention': (
            evidence.get('primary_evidence_class')
            == 'proposal_representation_or_supervision_bottleneck'
        ),
        'checkpoint_policy_requires_decoupling': as_bool(
            evidence.get(
                'checkpoint_selection_should_be_decoupled_from_patience',
                False,
            )
        ),
        'fold0_fit_has_82_subjects': len(fold0_fit_df) == 82,
        'fold0_heldout_has_20_subjects': len(fold0_heldout_df) == 20,
        'locked_validation_has_23_subjects': len(locked_validation_df) == 23,
        'sealed_test_has_23_subjects': len(sealed_test_df) == 23,
        'global_downsample_matches_r6_contract': (
            tuple(int(v) for v in GLOBAL_DOWNSAMPLE) == (2, 2, 2)
        ),
        'global_shape_matches_r6_contract': (
            tuple(int(v) for v in GLOBAL_INPUT_SHAPE) == (144, 160, 64)
        ),
        'architecture_parameter_count_is_unchanged': (
            int(EXPECTED_V22_PARAMETERS)
            == R6_EXPECTED_ARCHITECTURE_PARAMETERS
        ),
        'manual_r5_diagnostic_review_accepted': (
            R6_MANUAL_R5_DIAGNOSTIC_REVIEW_ACCEPTED
        ),
        'r6_output_is_outside_r5_checkpoint_tree': not str(
            R6_PREFLIGHT_ROOT
        ).startswith(str(PILOT_ROOT)),
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R6 UPSTREAM CONTRACT FAILED. Do not continue. Failed: '
            + ', '.join(failed)
        )
    return r5_result, fusion_gate, checks


def r6_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


def r6_unique_linear_coordinates(coordinates, pad_before):
    coordinates = np.asarray(coordinates, dtype=np.int64)
    pad_before = np.asarray(pad_before, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pooled = (coordinates + pad_before[None]) // downsample[None]
    if len(pooled) == 0:
        return np.empty(0, dtype=np.int64)
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        if int(pooled[:, axis].min()) < 0 or int(pooled[:, axis].max()) >= size:
            raise RuntimeError('Pooled component coordinate is out of bounds.')
    linear = np.ravel_multi_index(pooled.T, GLOBAL_INPUT_SHAPE)
    return np.unique(linear.astype(np.int64))


def r6_dense_mask(linear_coordinates, shape=GLOBAL_INPUT_SHAPE):
    mask = np.zeros(int(np.prod(shape)), dtype=bool)
    mask[np.asarray(linear_coordinates, dtype=np.int64)] = True
    return mask.reshape(shape)


def r6_sparse_max_pool(values, pad_before):
    values = np.asarray(values, dtype=np.float32)
    coordinates = np.argwhere(values > 0)
    if len(coordinates) == 0:
        return (
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.float32),
        )
    pad_before = np.asarray(pad_before, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pooled = (coordinates + pad_before[None]) // downsample[None]
    linear = np.ravel_multi_index(pooled.T, GLOBAL_INPUT_SHAPE)
    unique, inverse = np.unique(linear.astype(np.int64), return_inverse=True)
    maxima = np.zeros(len(unique), dtype=np.float32)
    np.maximum.at(
        maxima, inverse,
        values[tuple(coordinates[:, axis] for axis in range(3))],
    )
    return unique, maxima


def r6_dense_values(linear_coordinates, values, shape=GLOBAL_INPUT_SHAPE):
    dense = np.zeros(int(np.prod(shape)), dtype=np.float32)
    dense[np.asarray(linear_coordinates, dtype=np.int64)] = np.asarray(
        values, dtype=np.float32
    )
    return dense.reshape(shape)


def r6_audit_components():
    connectivity = np.ones((3, 3, 3), dtype=bool)
    expected = component_df.copy()
    expected['subject_id'] = expected['subject_id'].astype(str)
    expected_by_subject = {
        subject_id: group.sort_values('atomic_component_id')
        for subject_id, group in expected.groupby('subject_id')
    }
    component_pool = {}
    subject_target_sparse = {}
    rows = []
    target_access_rows = []
    fit_rows = fold0_fit_df.copy()
    fit_rows['subject_id'] = fit_rows['subject_id'].astype(str)
    fit_subjects = set(fit_rows['subject_id'])
    for row in fit_rows.sort_values('subject_id').to_dict('records'):
        subject_id = str(row['subject_id'])
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        with np.load(npz_path, allow_pickle=False) as data:
            required = {'target_binary', 'target_soft', 'target_spacing'}
            missing = sorted(required - set(data.files))
            if missing:
                raise RuntimeError(
                    f'{subject_id}: target arrays missing {missing}.'
                )
            binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
            soft = np.asarray(data['target_soft'], dtype=np.float32)
            spacing = np.asarray(
                data['target_spacing'], dtype=np.float64
            ).reshape(-1)
        if binary.shape != soft.shape:
            raise RuntimeError(f'{subject_id}: binary/soft shapes disagree.')
        if (
            spacing.shape != (3,)
            or not np.isfinite(spacing).all()
            or bool((spacing <= 0).any())
        ):
            raise RuntimeError(f'{subject_id}: target spacing is invalid.')
        if (
            not np.isfinite(soft).all()
            or bool((soft < 0).any())
            or bool((soft > 1).any())
        ):
            raise RuntimeError(f'{subject_id}: soft target is invalid.')

        supervision_mode = str(row['trial8_supervision_mode'])
        if supervision_mode == 'hard_and_soft':
            support = binary
        elif supervision_mode == 'soft_only_subvoxel':
            support = soft > 0
        elif supervision_mode == 'true_empty':
            support = np.zeros_like(binary, dtype=bool)
        else:
            raise RuntimeError(
                f'{subject_id}: unknown supervision mode '
                f'{supervision_mode!r}.'
            )
        cached_union = pilot_unpack_mask(
            pilot_mask_records[subject_id]['support']
        ).astype(bool)
        raw_union = binary | (soft > 0)
        if not np.array_equal(cached_union, raw_union):
            raise RuntimeError(
                f'{subject_id}: r5 cached target union does not match source.'
            )
        labels, count = ndimage.label(support, structure=connectivity)
        expected_subject = expected_by_subject.get(
            subject_id, pd.DataFrame(columns=expected.columns)
        )
        if count != len(expected_subject):
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {len(expected_subject)}.'
            )
        pad_total = (
            np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64)
            - np.asarray(support.shape, dtype=np.int64)
        )
        if bool((pad_total < 0).any()):
            raise RuntimeError(f'{subject_id}: support exceeds global canvas.')
        pad_before = pad_total // 2
        full_target = np.maximum(binary.astype(np.float32), soft)
        full_linear, full_values = r6_sparse_max_pool(
            full_target, pad_before
        )
        subject_target_sparse[subject_id] = (full_linear, full_values)
        target_access_rows.append({
            'subject_id': subject_id,
            'development_partition': 'fold0_fit',
            'supervision_mode': supervision_mode,
            'npz_path': str(npz_path),
            'binary_voxels': int(binary.sum()),
            'soft_support_voxels': int((soft > 0).sum()),
            'frozen_component_support_voxels': int(support.sum()),
            'global_target_nonzero_voxels': int(len(full_linear)),
            'spacing_x_mm': float(spacing[0]),
            'spacing_y_mm': float(spacing[1]),
            'spacing_z_mm': float(spacing[2]),
            'target_arrays_read': True,
            'fold0_heldout_target_read': False,
            'locked_validation_target_read': False,
            'test_target_read': False,
        })
        for component_id in range(1, count + 1):
            frozen = expected_subject[
                expected_subject['atomic_component_id'].astype(int)
                == component_id
            ]
            if len(frozen) != 1:
                raise RuntimeError(
                    f'{subject_id}: missing or duplicate component '
                    f'{component_id} in frozen audit.'
                )
            frozen = frozen.iloc[0]
            coordinates = np.argwhere(labels == component_id)
            pooled_linear = r6_unique_linear_coordinates(
                coordinates, pad_before
            )
            component_pool[(subject_id, component_id)] = pooled_linear
            native_voxels_match = (
                len(coordinates) == int(frozen['component_voxels'])
            )
            pooled_voxels_match = (
                len(pooled_linear)
                == int(frozen['global_proposal_voxels'])
            )
            volume_per_voxel_ml = (
                float(frozen['component_volume_ml'])
                / max(int(frozen['component_voxels']), 1)
            )
            rows.append({
                'subject_id': subject_id,
                'atomic_component_id': component_id,
                'component_tier': str(frozen['component_tier']),
                'native_component_voxels': int(len(coordinates)),
                'global_component_voxels': int(len(pooled_linear)),
                'component_volume_ml': float(
                    frozen['component_volume_ml']
                ),
                'volume_per_native_voxel_ml': volume_per_voxel_ml,
                'native_voxels_match_frozen_audit': native_voxels_match,
                'global_voxels_match_frozen_audit': pooled_voxels_match,
                'preserved_on_global_grid': len(pooled_linear) > 0,
                'development_partition': 'fold0_fit',
                'target_array_opened_by_r6_block': True,
            })

    audit = pd.DataFrame(rows)
    target_access = pd.DataFrame(target_access_rows)
    if len(audit):
        full_counts = {
            subject_id: len(values[0])
            for subject_id, values in subject_target_sparse.items()
        }
        audit['full_subject_global_positive_voxels'] = audit[
            'subject_id'
        ].map(full_counts).astype(int)
        audit['current_voxel_weighted_mass_fraction'] = (
            audit['global_component_voxels']
            / audit['full_subject_global_positive_voxels'].clip(lower=1)
        )
        audit['r6_selected_component_loss_units'] = 1.0
    return audit, component_pool, subject_target_sparse, target_access


def r6_audit_schedule(component_pool):
    rows = []
    heldout_subjects = set(fold0_heldout_df['subject_id'].astype(str))
    for epoch in (0, 1):
        first = pilot_build_epoch_schedule(epoch)
        second = pilot_build_epoch_schedule(epoch)
        if first != second:
            raise RuntimeError(f'Epoch {epoch} schedule is not deterministic.')
        for schedule_index, entry in enumerate(first):
            subject_id = str(entry['subject_id'])
            sample_type = str(entry['sample_type'])
            component_id = int(entry['atomic_component_id'])
            valid = sample_type == 'positive'
            key = (subject_id, component_id)
            mapped = key in component_pool if valid else component_id == 0
            component_voxels = (
                int(len(component_pool[key])) if valid and mapped else 0
            )
            rows.append({
                'epoch': int(epoch),
                'schedule_index': int(schedule_index),
                'sample_type': sample_type,
                'subject_id': subject_id,
                'atomic_component_id': component_id,
                'component_tier': str(entry['component_tier']),
                'component_volume_ml': float(entry['component_volume_ml']),
                'component_coverage_loss_valid': valid,
                'selected_component_maps_to_global_grid': bool(mapped),
                'selected_global_component_voxels': component_voxels,
                'subject_is_fold0_heldout': subject_id in heldout_subjects,
            })
    return pd.DataFrame(rows)


def r6_component_halo(exact_target):
    exact = np.asarray(exact_target, dtype=bool)
    structure = r6_physical_ball(
        R6_GLOBAL_SPACING_MM, R6_TOLERANCE_RADIUS_MM
    )
    return ndimage.binary_dilation(exact, structure=structure)


def r6_component_coverage_loss(
    logits, exact_component_target, tolerant_component_target,
    component_valid,
):
    probability = torch.sigmoid(logits.float())
    exact_target = exact_component_target.bool()
    tolerant_target = tolerant_component_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    exact_losses = []
    tolerant_losses = []
    exact_scores = []
    tolerant_scores = []
    for batch_index in range(probability.shape[0]):
        if not bool(valid[batch_index]):
            continue
        exact_values = probability[batch_index][exact_target[batch_index]]
        tolerant_values = probability[batch_index][
            tolerant_target[batch_index]
        ]
        if exact_values.numel() == 0 or tolerant_values.numel() == 0:
            raise RuntimeError(
                'A valid selected component has an empty exact/tolerant target.'
            )
        exact_score = exact_values.mean().clamp(1e-6, 1.0 - 1e-6)
        topk = min(
            R6_TOLERANT_TOPK,
            int(exact_values.numel()),
            int(tolerant_values.numel()),
        )
        tolerant_score = torch.topk(
            tolerant_values, k=max(topk, 1), largest=True
        ).values.mean().clamp(1e-6, 1.0 - 1e-6)
        exact_loss = -torch.log(exact_score)
        tolerant_loss = -torch.log(tolerant_score)
        loss = (
            R6_EXACT_COVERAGE_WEIGHT * exact_loss
            + R6_TOLERANT_COVERAGE_WEIGHT * tolerant_loss
        )
        losses.append(loss)
        exact_losses.append(exact_loss)
        tolerant_losses.append(tolerant_loss)
        exact_scores.append(exact_score)
        tolerant_scores.append(tolerant_score)
    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'selected_component_coverage_loss': zero,
            'selected_component_exact_loss': zero,
            'selected_component_tolerant_loss': zero,
            'selected_component_exact_score': zero.detach(),
            'selected_component_tolerant_score': zero.detach(),
            'selected_components_in_batch': 0,
        }
    return torch.stack(losses).mean(), {
        'selected_component_coverage_loss': torch.stack(losses).mean(),
        'selected_component_exact_loss': torch.stack(exact_losses).mean(),
        'selected_component_tolerant_loss': torch.stack(
            tolerant_losses
        ).mean(),
        'selected_component_exact_score': torch.stack(exact_scores).mean(),
        'selected_component_tolerant_score': torch.stack(
            tolerant_scores
        ).mean(),
        'selected_components_in_batch': len(losses),
    }


def r6_current_proposal_loss(logits, full_target):
    dice = pilot_soft_dice_loss(logits, full_target).mean()
    focal_bce = pilot_focal_bce_loss(logits, full_target).mean()
    return 0.65 * dice + 0.35 * focal_bce, {
        'full_proposal_dice_loss': dice,
        'full_proposal_focal_bce_loss': focal_bce,
    }


def r6_component_preserving_proposal_loss(
    logits, full_target, exact_component_target,
    tolerant_component_target, component_valid,
):
    current, current_parts = r6_current_proposal_loss(logits, full_target)
    coverage, coverage_parts = r6_component_coverage_loss(
        logits, exact_component_target, tolerant_component_target,
        component_valid,
    )
    total = current + R6_COMPONENT_LOSS_COEFFICIENT * coverage
    return total, {
        **current_parts,
        **coverage_parts,
        'current_whole_mask_proposal_loss': current,
        'component_loss_coefficient': R6_COMPONENT_LOSS_COEFFICIENT,
        'component_preserving_proposal_loss': total,
    }


def r6_gradient_record(
    logits_shape, full_target_np, exact_target_np, tier, label,
):
    full_target = torch.from_numpy(
        np.asarray(full_target_np, dtype=np.float32)[None, None]
    )
    exact_target = torch.from_numpy(
        np.asarray(exact_target_np, dtype=bool)[None, None]
    )
    halo_np = r6_component_halo(exact_target_np)
    tolerant_target = torch.from_numpy(halo_np[None, None])
    valid = torch.ones(1, dtype=torch.bool)

    old_logits = torch.full(
        logits_shape, -2.0, dtype=torch.float32, requires_grad=True
    )
    old_loss, _ = r6_current_proposal_loss(old_logits, full_target)
    old_gradient = torch.autograd.grad(old_loss, old_logits)[0].detach()

    new_logits = torch.full(
        logits_shape, -2.0, dtype=torch.float32, requires_grad=True
    )
    new_loss, parts = r6_component_preserving_proposal_loss(
        new_logits, full_target, exact_target, tolerant_target, valid
    )
    new_gradient = torch.autograd.grad(new_loss, new_logits)[0].detach()

    exact = exact_target.bool()
    halo = tolerant_target.bool()
    outside_halo = ~halo
    old_selected = float(old_gradient[exact].abs().sum())
    new_selected = float(new_gradient[exact].abs().sum())
    outside_delta = (
        new_gradient[outside_halo] - old_gradient[outside_halo]
    ).abs()
    outside_difference = (
        float(outside_delta.max()) if outside_delta.numel() else 0.0
    )
    record = {
        'probe': str(label),
        'component_tier': str(tier),
        'component_global_voxels': int(exact.sum()),
        'tolerant_global_voxels': int(halo.sum()),
        'current_loss': float(old_loss.detach()),
        'r6_loss': float(new_loss.detach()),
        'coverage_loss': float(
            parts['selected_component_coverage_loss'].detach()
        ),
        'current_selected_gradient_l1': old_selected,
        'r6_selected_gradient_l1': new_selected,
        'selected_gradient_amplification': (
            new_selected / max(old_selected, 1e-12)
        ),
        'current_total_gradient_l1': float(old_gradient.abs().sum()),
        'r6_total_gradient_l1': float(new_gradient.abs().sum()),
        'maximum_gradient_change_outside_tolerant_region': outside_difference,
        'all_gradients_finite': bool(
            torch.isfinite(old_gradient).all()
            and torch.isfinite(new_gradient).all()
        ),
    }
    del old_logits, new_logits, old_gradient, new_gradient
    gc.collect()
    return record


def r6_synthetic_gradient_audit():
    shape = (32, 32, 16)
    large = np.zeros(shape, dtype=bool)
    large[2:10, 2:10, 2:6] = True
    candidate = np.argwhere(
        np.pad(
            np.ones((10, 10, 7), dtype=bool),
            ((20, 2), (20, 2), (8, 1)),
            mode='constant', constant_values=False,
        )
    )
    rows = []
    for size in R6_SYNTHETIC_COMPONENT_SIZES:
        exact = np.zeros(shape, dtype=bool)
        coordinates = candidate[:size]
        exact[tuple(coordinates[:, axis] for axis in range(3))] = True
        full = large | exact
        rows.append(r6_gradient_record(
            (1, 1, *shape), full, exact,
            tier=f'synthetic_{size}_voxels', label=f'synthetic_{size}',
        ))
    return pd.DataFrame(rows)


def r6_representative_components(component_audit):
    rows = []
    for tier, group in component_audit.groupby('component_tier'):
        ordered = group.sort_values([
            'global_component_voxels', 'subject_id', 'atomic_component_id'
        ]).reset_index(drop=True)
        rows.append(ordered.iloc[len(ordered) // 2])
    return pd.DataFrame(rows).reset_index(drop=True)


def r6_real_gradient_audit(
    component_audit, component_pool, subject_target_sparse,
):
    representatives = r6_representative_components(component_audit)
    rows = []
    for record in representatives.to_dict('records'):
        subject_id = str(record['subject_id'])
        component_id = int(record['atomic_component_id'])
        full_linear, full_values = subject_target_sparse[subject_id]
        full = r6_dense_values(full_linear, full_values)
        exact = r6_dense_mask(component_pool[(subject_id, component_id)])
        row = r6_gradient_record(
            (1, 1, *GLOBAL_INPUT_SHAPE), full, exact,
            tier=str(record['component_tier']),
            label=f'{subject_id}_component_{component_id}',
        )
        row.update({
            'subject_id': subject_id,
            'atomic_component_id': component_id,
            'component_volume_ml': float(record['component_volume_ml']),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def r6_tier_summary(component_audit, schedule_audit):
    component_summary = component_audit.groupby(
        'component_tier', as_index=False
    ).agg(
        components=('atomic_component_id', 'count'),
        subjects=('subject_id', 'nunique'),
        minimum_global_voxels=('global_component_voxels', 'min'),
        median_global_voxels=('global_component_voxels', 'median'),
        maximum_global_voxels=('global_component_voxels', 'max'),
        median_current_voxel_mass_fraction=(
            'current_voxel_weighted_mass_fraction', 'median'
        ),
        maximum_current_voxel_mass_fraction=(
            'current_voxel_weighted_mass_fraction', 'max'
        ),
    )
    selected = schedule_audit[
        schedule_audit['component_coverage_loss_valid']
    ].groupby('component_tier').size().rename('two_epoch_selections')
    return component_summary.merge(
        selected, left_on='component_tier', right_index=True, how='left'
    ).fillna({'two_epoch_selections': 0})


def r6_display(frame):
    if 'display' in globals():
        display(frame)
    else:
        print(frame.to_string(index=False))


def run_r6_component_proposal_preflight():
    start_time = time.time()
    r5_result, fusion_gate, runtime_checks = r6_require_runtime_contract()
    if R6_PREFLIGHT_GATE_PATH.exists():
        completed = load_json(R6_PREFLIGHT_GATE_PATH)
        if (
            completed.get('preflight_version') == R6_PREFLIGHT_VERSION
            and completed.get('code_revision') == R6_PREFLIGHT_CODE_REVISION
            and as_bool(completed.get('preflight_passed', False))
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R6 COMPONENT-PROPOSAL PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {R6_PREFLIGHT_GATE_PATH}')
            print('No r6 artifact was rewritten.')
            return
        raise RuntimeError(
            'An incompatible or incomplete artifact already occupies the r6 '
            'preflight directory. Refusing to overwrite it.'
        )
    required_paths = [
        Path(PILOT_RESULT_GATE_PATH), R5_FUSION_GATE_PATH,
    ]
    source_hashes_before = {
        str(path): file_sha256(path) for path in required_paths
    }

    (
        component_audit, component_pool,
        subject_target_sparse, target_access_audit,
    ) = r6_audit_components()
    schedule_audit = r6_audit_schedule(component_pool)
    synthetic_gradient = r6_synthetic_gradient_audit()
    real_gradient = r6_real_gradient_audit(
        component_audit, component_pool, subject_target_sparse
    )
    tier_summary = r6_tier_summary(component_audit, schedule_audit)

    source_hashes_after = {
        str(path): file_sha256(path) for path in required_paths
    }
    positive_schedule = schedule_audit[
        schedule_audit['component_coverage_loss_valid']
    ]
    negative_schedule = schedule_audit[
        ~schedule_audit['component_coverage_loss_valid']
    ]
    expected_voxel_volume_ml = float(
        np.prod(R6_NATIVE_SPACING_MM) / 1000.0
    )
    synthetic_ratio_current = float(
        synthetic_gradient['current_selected_gradient_l1'].max()
        / synthetic_gradient['current_selected_gradient_l1'].min()
    )
    synthetic_ratio_r6 = float(
        synthetic_gradient['r6_selected_gradient_l1'].max()
        / synthetic_gradient['r6_selected_gradient_l1'].min()
    )
    small_tiers = {
        'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
    }
    small_real = real_gradient[
        real_gradient['component_tier'].isin(small_tiers)
    ]

    checks = {
        **runtime_checks,
        'component_audit_has_872_atomic_components': (
            len(component_audit) == R6_EXPECTED_ATOMIC_COMPONENTS
        ),
        'all_component_subjects_are_fold0_fit': set(
            component_audit['subject_id']
        ).issubset(set(fold0_fit_df['subject_id'].astype(str))),
        'native_component_counts_match_frozen_audit': bool(
            component_audit[
                'native_voxels_match_frozen_audit'
            ].all()
        ),
        'global_component_counts_match_frozen_audit': bool(
            component_audit[
                'global_voxels_match_frozen_audit'
            ].all()
        ),
        'all_components_preserved_on_global_grid': bool(
            component_audit['preserved_on_global_grid'].all()
        ),
        'fixed_spacing_matches_component_volume_metadata': bool(
            np.allclose(
                component_audit['volume_per_native_voxel_ml'],
                expected_voxel_volume_ml, rtol=0.0, atol=1e-7,
            )
        ),
        'source_target_spacing_matches_r6_contract': bool(np.allclose(
            target_access_audit[[
                'spacing_x_mm', 'spacing_y_mm', 'spacing_z_mm'
            ]].to_numpy(dtype=np.float64),
            np.asarray(R6_NATIVE_SPACING_MM, dtype=np.float64)[None],
            rtol=0.0, atol=1e-6,
        )),
        'two_epoch_schedule_has_336_samples': len(schedule_audit) == 336,
        'two_epoch_schedule_has_224_positive_samples': (
            len(positive_schedule) == 224
        ),
        'two_epoch_schedule_has_112_negative_samples': (
            len(negative_schedule) == 112
        ),
        'all_positive_schedule_components_map_to_global_grid': bool(
            positive_schedule[
                'selected_component_maps_to_global_grid'
            ].all()
            and (positive_schedule[
                'selected_global_component_voxels'
            ] > 0).all()
        ),
        'negative_samples_do_not_receive_component_loss': bool(
            (~negative_schedule['component_coverage_loss_valid']).all()
            and (negative_schedule[
                'selected_global_component_voxels'
            ] == 0).all()
        ),
        'heldout_subjects_not_sampled': bool(
            (~schedule_audit['subject_is_fold0_heldout']).all()
        ),
        'all_component_tiers_receive_r6_supervision': set(
            positive_schedule['component_tier']
        ) == set(component_audit['component_tier']),
        'synthetic_gradients_are_finite': bool(
            synthetic_gradient['all_gradients_finite'].all()
        ),
        'real_gradients_are_finite': bool(
            real_gradient['all_gradients_finite'].all()
        ),
        'r6_equalizes_synthetic_component_gradient_mass': (
            synthetic_ratio_r6 < synthetic_ratio_current
            and synthetic_ratio_r6 <= 2.0
        ),
        'r6_amplifies_every_small_tier_probe': bool(
            len(small_real) == 4
            and (small_real['selected_gradient_amplification'] > 1.0).all()
        ),
        'whole_mask_gradient_is_unchanged_outside_tolerant_region': bool(
            (synthetic_gradient[
                'maximum_gradient_change_outside_tolerant_region'
            ] <= 1e-12).all()
            and (real_gradient[
                'maximum_gradient_change_outside_tolerant_region'
            ] <= 1e-12).all()
        ),
        'r5_gate_hashes_unchanged': source_hashes_before == source_hashes_after,
        'exactly_82_fold0_fit_targets_read': (
            len(target_access_audit) == 82
            and target_access_audit['subject_id'].nunique() == 82
        ),
        'target_reads_are_exactly_fold0_fit': set(
            target_access_audit['subject_id']
        ) == set(fold0_fit_df['subject_id'].astype(str)),
        'all_component_targets_rebuilt_from_source': bool(
            component_audit['target_array_opened_by_r6_block'].all()
        ),
        'target_access_audit_reports_no_heldout_reads': bool(
            (~target_access_audit['fold0_heldout_target_read']).all()
        ),
        'target_access_audit_reports_no_locked_validation_reads': bool(
            (~target_access_audit['locked_validation_target_read']).all()
        ),
        'target_access_audit_reports_no_test_reads': bool(
            (~target_access_audit['test_target_read']).all()
        ),
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'outcome_workbooks_not_read': True,
        'study_model_not_constructed': True,
        'study_trainable_parameters_are_zero': True,
        'optimizer_was_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'r5_artifacts_not_modified': source_hashes_before == source_hashes_after,
        'r6_pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        print('\nR6 preflight checks:')
        r6_display(pd.DataFrame({
            'check': list(checks), 'passed': list(checks.values())
        }))
        raise RuntimeError(
            'TRIAL8 R6 COMPONENT-PROPOSAL PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': R6_PREFLIGHT_VERSION,
        'code_revision': R6_PREFLIGHT_CODE_REVISION,
        'parent_pilot_version': EXPECTED_R5_PILOT_VERSION,
        'architecture_parameters': int(EXPECTED_V22_PARAMETERS),
        'new_trainable_parameters': 0,
        'fresh_start_from_initialization': True,
        'whole_mask_proposal_objective_preserved': {
            'soft_dice_weight': 0.65,
            'focal_bce_weight': 0.35,
        },
        'selected_component_objective': {
            'coefficient_added_to_whole_mask_loss': (
                R6_COMPONENT_LOSS_COEFFICIENT
            ),
            'exact_coverage_weight': R6_EXACT_COVERAGE_WEIGHT,
            'tolerant_coverage_weight': R6_TOLERANT_COVERAGE_WEIGHT,
            'tolerance_radius_mm': R6_TOLERANCE_RADIUS_MM,
            'tolerant_topk': R6_TOLERANT_TOPK,
            'valid_samples': 'positive component-balanced schedule entries only',
            'negative_samples': 'whole-mask proposal objective only',
        },
        'outer_training_loss_proposal_weight': 0.35,
        'checkpoint_policy': {
            'save_checkpoint_on': 'every strict numeric score improvement',
            'minimum_delta_use': 'early-stopping patience reset only',
            'minimum_delta_must_not_block_checkpoint_save': True,
        },
        'r5_baseline': fusion_gate['evidence_summary'],
        'r6_directional_success_gates': {
            'mean_dice_absolute_improvement_at_least': 0.015,
            'small_component_within_8mm_absolute_improvement_at_least': 0.10,
            'small_representation_failure_absolute_reduction_at_least': 0.10,
            'proposal_component_recall_minimum': 0.12,
            'global_dice_maximum_allowed_regression': 0.02,
            'mean_false_positive_volume_ml_maximum': 30.0,
            'large_component_proposal_recall_minimum': 0.83,
        },
        'fold0_heldout_targets_available_to_training': False,
        'locked_validation_targets_available_to_training': False,
        'test_targets_available': False,
        'r6_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'manual_r5_diagnostic_review_accepted': (
            R6_MANUAL_R5_DIAGNOSTIC_REVIEW_ACCEPTED
        ),
        'target_access_policy': {
            'authorized_split': 'development train / fold0 fit only',
            'fit_subjects_read': 82,
            'fold0_heldout_targets_read': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        },
    }

    R6_PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(component_audit, R6_COMPONENT_AUDIT_PATH)
    atomic_write_csv(target_access_audit, R6_TARGET_ACCESS_AUDIT_PATH)
    atomic_write_csv(schedule_audit, R6_SCHEDULE_AUDIT_PATH)
    atomic_write_csv(synthetic_gradient, R6_SYNTHETIC_GRADIENT_PATH)
    atomic_write_csv(real_gradient, R6_REAL_GRADIENT_PATH)
    atomic_write_csv(tier_summary, R6_TIER_SUMMARY_PATH)
    atomic_write_json(policy, R6_POLICY_PATH)

    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': R6_PREFLIGHT_VERSION,
        'code_revision': R6_PREFLIGHT_CODE_REVISION,
        'preflight_passed': True,
        'checks': checks,
        'source_gate_sha256': source_hashes_before,
        'components_audited': int(len(component_audit)),
        'schedule_samples_audited': int(len(schedule_audit)),
        'synthetic_gradient_probes': int(len(synthetic_gradient)),
        'real_gradient_probes': int(len(real_gradient)),
        'current_synthetic_gradient_mass_ratio': synthetic_ratio_current,
        'r6_synthetic_gradient_mass_ratio': synthetic_ratio_r6,
        'architecture_parameters': int(EXPECTED_V22_PARAMETERS),
        'new_trainable_parameters': 0,
        'target_arrays_opened_by_r6_block': True,
        'target_arrays_read_splits': ['fold0_fit'],
        'target_subjects_read': int(len(target_access_audit)),
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'study_trainable_parameters': 0,
        'optimizer_steps': 0,
        'training_performed': False,
        'checkpoint_written': False,
        'r6_pilot_implementation_allowed': True,
        'r6_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'policy_path': str(R6_POLICY_PATH),
        'component_audit_path': str(R6_COMPONENT_AUDIT_PATH),
        'target_access_audit_path': str(R6_TARGET_ACCESS_AUDIT_PATH),
        'schedule_audit_path': str(R6_SCHEDULE_AUDIT_PATH),
        'synthetic_gradient_audit_path': str(R6_SYNTHETIC_GRADIENT_PATH),
        'real_gradient_audit_path': str(R6_REAL_GRADIENT_PATH),
        'tier_summary_path': str(R6_TIER_SUMMARY_PATH),
        'elapsed_seconds': float(time.time() - start_time),
    }
    atomic_write_json(gate, R6_PREFLIGHT_GATE_PATH)

    print('\nComponent supervision by tier:')
    r6_display(tier_summary)
    print('\nSynthetic gradient-allocation audit:')
    r6_display(synthetic_gradient)
    print('\nReal representative-component gradient audit:')
    r6_display(real_gradient)
    print('\nR6 preflight checks:')
    r6_display(pd.DataFrame({
        'check': list(checks), 'passed': list(checks.values())
    }))

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R6 COMPONENT-PRESERVING PROPOSAL PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Fold-0 fit subjects:                 {len(fold0_fit_df)}')
    print(f'Atomic components audited:          {len(component_audit)}')
    print(f'Two-epoch schedule samples:         {len(schedule_audit)}')
    print('Architecture parameters:            '
          f'{EXPECTED_V22_PARAMETERS:,} (unchanged)')
    print('New trainable parameters:            0')
    print('Whole-mask proposal loss:            preserved exactly')
    print('Selected-component coefficient:      '
          f'{R6_COMPONENT_LOSS_COEFFICIENT:.2f}')
    print('Physical tolerance radius:           '
          f'{R6_TOLERANCE_RADIUS_MM:.1f} mm')
    print('Fold-0 fit targets read for audit:    82')
    print('Fold-0 held-out targets read:         False')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Training performed:                   False')
    print('R6 pilot implementation allowed:      True')
    print('R6 pilot training authorized:         False')
    print('Full five-fold training authorized:   False')
    print(f'Policy:                              {R6_POLICY_PATH}')
    print(f'Preflight gate:                      {R6_PREFLIGHT_GATE_PATH}')
    print('\nNext: build the isolated fresh-start fold-0 r6 pilot training '
          'block from this frozen loss contract; do not launch five folds.')


run_r6_component_proposal_preflight()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = (
    'r6_component_equalized_proposal_strict_best_r2_smoke'
)
PILOT_VERSION = 'v2_2_fold0_component_preserving_proposal_r6'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261003
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

R6_PREFLIGHT_VERSION = 'v1_equalized_selected_component_coverage'
R6_PREFLIGHT_CODE_REVISION = 'r2_exact_frozen_support_policy'
R6_COMPONENT_LOSS_COEFFICIENT = 0.25
R6_EXACT_COVERAGE_WEIGHT = 0.80
R6_TOLERANT_COVERAGE_WEIGHT = 0.20
R6_TOLERANCE_RADIUS_MM = 4.0
R6_TOLERANT_TOPK = 4
R6_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip((0.8, 0.8, 2.0), GLOBAL_DOWNSAMPLE)
)

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'
R6_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r6_component_preserving_proposal' / R6_PREFLIGHT_VERSION
)
R6_PREFLIGHT_GATE_PATH = (
    R6_PREFLIGHT_ROOT / 'r6_component_proposal_preflight_gate.json'
)
R6_POLICY_PATH = R6_PREFLIGHT_ROOT / 'r6_proposal_loss_policy.json'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH, R6_PREFLIGHT_GATE_PATH, R6_POLICY_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
r6_preflight_gate = load_json(R6_PREFLIGHT_GATE_PATH)
r6_policy = load_json(R6_POLICY_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r6_component_preflight_passed': (
        r6_preflight_gate.get('preflight_passed') is True
    ),
    'r6_component_preflight_all_checks_true': (
        len(r6_preflight_gate.get('checks', {})) > 0
        and all(bool(value) for value in r6_preflight_gate['checks'].values())
    ),
    'r6_preflight_version_matches': (
        r6_preflight_gate.get('preflight_version') == R6_PREFLIGHT_VERSION
    ),
    'r6_preflight_revision_matches': (
        r6_preflight_gate.get('code_revision')
        == R6_PREFLIGHT_CODE_REVISION
    ),
    'r6_implementation_allowed': (
        r6_preflight_gate.get('r6_pilot_implementation_allowed') is True
    ),
    'r6_training_was_locked_before_this_block': (
        r6_preflight_gate.get('r6_pilot_training_authorized') is False
    ),
    'r6_architecture_parameter_count_unchanged': (
        int(r6_preflight_gate.get('architecture_parameters', -1))
        == EXPECTED_V22_PARAMETERS
        and int(r6_preflight_gate.get('new_trainable_parameters', -1)) == 0
    ),
    'r6_preflight_target_scope_is_fit_only': (
        r6_preflight_gate.get('target_arrays_read_splits') == ['fold0_fit']
        and r6_preflight_gate.get('fold0_heldout_targets_read') is False
        and r6_preflight_gate.get('locked_validation_targets_read') is False
        and r6_preflight_gate.get('test_targets_read') is False
    ),
    'r6_policy_path_matches_gate': (
        str(r6_preflight_gate.get('policy_path')) == str(R6_POLICY_PATH)
    ),
    'r6_policy_component_coefficient_matches': math.isclose(
        float(r6_policy.get('selected_component_objective', {}).get(
            'coefficient_added_to_whole_mask_loss', float('nan')
        )),
        R6_COMPONENT_LOSS_COEFFICIENT,
        rel_tol=0.0,
        abs_tol=1e-12,
    ),
    'r6_policy_preserves_whole_mask_loss': (
        math.isclose(
            float(r6_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('soft_dice_weight', float('nan'))),
            0.65, rel_tol=0.0, abs_tol=1e-12,
        )
        and math.isclose(
            float(r6_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('focal_bce_weight', float('nan'))),
            0.35, rel_tol=0.0, abs_tol=1e-12,
        )
    ),
    'r6_checkpoint_policy_is_decoupled': (
        r6_policy.get('checkpoint_policy', {}).get('save_checkpoint_on')
        == 'every strict numeric score improvement'
        and r6_policy.get('checkpoint_policy', {}).get('minimum_delta_use')
        == 'early-stopping patience reset only'
        and r6_policy.get('checkpoint_policy', {}).get(
            'minimum_delta_must_not_block_checkpoint_save'
        ) is True
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def r6_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


R6_TOLERANCE_STRUCTURE = r6_physical_ball(
    R6_GLOBAL_SPACING_MM, R6_TOLERANCE_RADIUS_MM
)


def r6_component_halo(exact_target):
    exact_target = np.asarray(exact_target, dtype=bool)
    return ndimage.binary_dilation(
        exact_target, structure=R6_TOLERANCE_STRUCTURE
    )


def pilot_extract_targets(
    row, local_origin, sample_type, atomic_component_id
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary > 0
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)

    selected_valid = str(sample_type) == 'positive'
    selected_component = np.zeros_like(component_support, dtype=bool)
    if selected_valid:
        labels, count = ndimage.label(
            component_support, structure=np.ones((3, 3, 3), dtype=bool)
        )
        expected_count = len(component_records.get(subject_id, []))
        if count != expected_count:
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {expected_count}.'
            )
        component_id = int(atomic_component_id)
        if component_id < 1 or component_id > count:
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is outside '
                f'the frozen 1..{count} range.'
            )
        selected_component = labels == component_id
        if not bool(selected_component.any()):
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is empty.'
            )

    selected_canvas, selected_start = symmetric_pad(
        selected_component.astype(np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if selected_start != start_a:
        raise RuntimeError(
            f'{subject_id}: selected-component padding start disagrees.'
        )
    selected_global = F.max_pool3d(
        torch.from_numpy(selected_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0
    selected_tolerant = (
        r6_component_halo(selected_global)
        if selected_valid else np.zeros_like(selected_global, dtype=bool)
    )
    if selected_valid:
        frozen = [
            item for item in component_records[subject_id]
            if int(item['atomic_component_id']) == int(atomic_component_id)
        ]
        if len(frozen) != 1:
            raise RuntimeError(
                f'{subject_id}: selected component does not map uniquely to '
                'the frozen component audit.'
            )
        expected_global_voxels = int(frozen[0]['global_proposal_voxels'])
        if int(selected_global.sum()) != expected_global_voxels:
            raise RuntimeError(
                f'{subject_id}: selected component has '
                f'{int(selected_global.sum())} pooled voxels; expected '
                f'{expected_global_voxels}.'
            )
        if not bool(selected_tolerant[selected_global].all()):
            raise RuntimeError(
                f'{subject_id}: tolerance halo does not contain exact target.'
            )
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'selected_component_target': selected_global[None].astype(bool),
        'selected_component_tolerant_target': (
            selected_tolerant[None].astype(bool)
        ),
        'selected_component_valid': np.asarray(
            selected_valid, dtype=bool
        ),
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'],
        )
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'selected_component_target': targets[
                'selected_component_target'
            ],
            'selected_component_tolerant_target': targets[
                'selected_component_tolerant_target'
            ],
            'selected_component_valid': targets[
                'selected_component_valid'
            ],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def r6_component_coverage_loss(
    logits, exact_component_target, tolerant_component_target,
    component_valid,
):
    probability = torch.sigmoid(logits.float())
    exact_target = exact_component_target.bool()
    tolerant_target = tolerant_component_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    exact_losses = []
    tolerant_losses = []
    exact_scores = []
    tolerant_scores = []
    for batch_index in range(probability.shape[0]):
        if not bool(valid[batch_index]):
            continue
        exact_values = probability[batch_index][exact_target[batch_index]]
        tolerant_values = probability[batch_index][
            tolerant_target[batch_index]
        ]
        if exact_values.numel() == 0 or tolerant_values.numel() == 0:
            raise RuntimeError(
                'A valid selected component has an empty exact/tolerant target.'
            )
        exact_score = exact_values.mean().clamp(1e-6, 1.0 - 1e-6)
        topk = min(
            R6_TOLERANT_TOPK,
            int(exact_values.numel()),
            int(tolerant_values.numel()),
        )
        tolerant_score = torch.topk(
            tolerant_values, k=max(topk, 1), largest=True
        ).values.mean().clamp(1e-6, 1.0 - 1e-6)
        exact_loss = -torch.log(exact_score)
        tolerant_loss = -torch.log(tolerant_score)
        coverage_loss = (
            R6_EXACT_COVERAGE_WEIGHT * exact_loss
            + R6_TOLERANT_COVERAGE_WEIGHT * tolerant_loss
        )
        losses.append(coverage_loss)
        exact_losses.append(exact_loss)
        tolerant_losses.append(tolerant_loss)
        exact_scores.append(exact_score)
        tolerant_scores.append(tolerant_score)

    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'selected_component_coverage_loss': zero,
            'selected_component_exact_loss': zero,
            'selected_component_tolerant_loss': zero,
            'selected_component_exact_score': zero.detach(),
            'selected_component_tolerant_score': zero.detach(),
            'selected_components_in_batch': zero.detach(),
        }
    return torch.stack(losses).mean(), {
        'selected_component_coverage_loss': torch.stack(losses).mean(),
        'selected_component_exact_loss': torch.stack(exact_losses).mean(),
        'selected_component_tolerant_loss': torch.stack(
            tolerant_losses
        ).mean(),
        'selected_component_exact_score': torch.stack(exact_scores).mean(),
        'selected_component_tolerant_score': torch.stack(
            tolerant_scores
        ).mean(),
        'selected_components_in_batch': torch.as_tensor(
            float(len(losses)), device=logits.device
        ),
    }


def r6_component_preserving_proposal_loss(
    logits, full_target, exact_component_target,
    tolerant_component_target, component_valid,
):
    whole_dice = pilot_soft_dice_loss(logits, full_target).mean()
    whole_bce = pilot_focal_bce_loss(logits, full_target).mean()
    whole_loss = 0.65 * whole_dice + 0.35 * whole_bce
    coverage_loss, coverage_parts = r6_component_coverage_loss(
        logits, exact_component_target, tolerant_component_target,
        component_valid,
    )
    proposal_loss = (
        whole_loss + R6_COMPONENT_LOSS_COEFFICIENT * coverage_loss
    )
    return proposal_loss, {
        'proposal_whole_mask_loss': whole_loss,
        'proposal_dice_loss': whole_dice,
        'proposal_focal_bce_loss': whole_bce,
        **coverage_parts,
    }


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(
    model, cpu_batch, device, epoch, batch_index,
    dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    if dropout_flags_override is not None:
        required_flags = {
            'force_atlas_dropout', 'force_vascular_dropout',
            'force_occlusion_dropout', 'force_clinical_dropout',
            'force_global_dropout', 'force_arterial_dropout',
        }
        if set(dropout_flags_override) != required_flags:
            raise RuntimeError(
                'Dropout override keys do not match the frozen branch set.'
            )
        flags = {
            name: bool(dropout_flags_override[name])
            for name in sorted(required_flags)
        }
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_loss, proposal_parts = r6_component_preserving_proposal_loss(
        output['global_proposal_logits'], global_target,
        batch['selected_component_target'],
        batch['selected_component_tolerant_target'],
        batch['selected_component_valid'],
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        **proposal_parts,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'arterial_valid_nodes': arterial_valid.sum().detach().float(),
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'arterial_dropped': torch.as_tensor(float(flags['force_arterial_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        proposal_distance = None
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                proposal_distance_mm = float('nan')
                if proposal_distance is not None:
                    proposal_distance_mm = float(
                        proposal_distance[component_slice][component_local].min(
                            initial=np.inf
                        )
                    )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'proposal_minimum_distance_mm': proposal_distance_mm,
                    'proposal_within_2mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 2.0
                    ),
                    'proposal_within_4mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 4.0
                    ),
                    'proposal_within_8mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 8.0
                    ),
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_components = [
            item for item in component_rows
            if item['component_tier'] == tier
        ]
        finite_distances = [
            float(item['proposal_minimum_distance_mm'])
            for item in tier_components
            if np.isfinite(item['proposal_minimum_distance_mm'])
        ]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
            'proposal_within_2mm_fraction': float(np.mean([
                item['proposal_within_2mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_4mm_fraction': float(np.mean([
                item['proposal_within_4mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_8mm_fraction': float(np.mean([
                item['proposal_within_8mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'median_finite_proposal_distance_mm': float(
                np.median(finite_distances)
            ) if finite_distances else float('nan'),
            'mean_proposal_maximum_probability': float(np.mean([
                item['proposal_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'mean_final_maximum_probability': float(np.mean([
                item['final_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'representation_failure_fraction': float(np.mean([
                item['representation_failure_below_0p10']
                for item in tier_components
            ])) if tier_components else 0.0,
        })
    small_component_rows = [
        item for item in component_rows
        if item['component_tier'] in set(COMPONENT_TIERS[:4])
    ]
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'small_components_audited': len(small_component_rows),
        'small_component_proposal_within_2mm_fraction': float(np.mean([
            item['proposal_within_2mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_4mm_fraction': float(np.mean([
            item['proposal_within_4mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_8mm_fraction': float(np.mean([
            item['proposal_within_8mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_representation_failure_fraction': float(np.mean([
            item['representation_failure_below_0p10']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R6 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_index = None
    smoke_subject_id = None
    for candidate_index, entry in enumerate(dataset.schedule):
        if entry['sample_type'] != 'positive':
            continue
        candidate_subject = str(entry['subject_id'])
        if not as_bool(
            fit_row_by_subject[candidate_subject]['trial8_atlas_enabled']
        ):
            continue
        candidate_sample = dataset[candidate_index]
        candidate_checks = {
            'atlas_enabled': bool(np.asarray(
                candidate_sample['atlas_enabled']
            ).reshape(-1)[0]),
            'territory_nodes_valid': bool(np.asarray(
                candidate_sample['territory_valid']
            ).astype(bool).any()),
            'vessel_nodes_valid': bool(np.asarray(
                candidate_sample['vessel_valid']
            ).astype(bool).any()),
            'selected_component_valid': bool(np.asarray(
                candidate_sample['selected_component_valid']
            ).reshape(-1)[0]),
        }
        if all(candidate_checks.values()):
            smoke_index = int(candidate_index)
            smoke_subject_id = candidate_subject
            break
    if smoke_index is None:
        raise RuntimeError(
            'No positive fold-0-fit sample has simultaneous atlas, territory, '
            'vessel, and selected-component support for the all-branch smoke '
            'test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    smoke_dropout_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0,
            dropout_flags_override=smoke_dropout_flags,
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    r6_smoke_checks = {
        'positive_sample_selected': bool(
            smoke_batch['selected_component_valid'].bool().all().item()
        ),
        'atlas_enabled_for_smoke': bool(
            smoke_batch['atlas_enabled'].bool().all().item()
        ),
        'territory_support_for_smoke': bool(
            smoke_batch['territory_valid'].bool().any().item()
        ),
        'vessel_support_for_smoke': bool(
            smoke_batch['vessel_valid'].bool().any().item()
        ),
        'projected_arterial_nodes_valid_for_smoke': float(
            smoke_parts['arterial_valid_nodes'].detach().float().cpu()
        ) > 0.0,
        'arterial_loss_finite_for_smoke': bool(torch.isfinite(
            smoke_parts['arterial_loss']
        ).item()),
        'all_branch_dropouts_disabled_for_smoke': not any(
            smoke_dropout_flags.values()
        ),
        'exact_component_target_nonempty': bool(
            smoke_batch['selected_component_target'].bool().any().item()
        ),
        'tolerant_target_contains_exact_target': bool(torch.all(
            ~smoke_batch['selected_component_target'].bool()
            | smoke_batch['selected_component_tolerant_target'].bool()
        ).item()),
        'selected_component_coverage_loss_positive': float(
            smoke_parts['selected_component_coverage_loss']
            .detach().float().cpu()
        ) > 0.0,
        'whole_mask_proposal_loss_finite': bool(torch.isfinite(
            smoke_parts['proposal_whole_mask_loss']
        ).item()),
    }
    if not all(r6_smoke_checks.values()):
        failed = [
            name for name, passed in r6_smoke_checks.items() if not passed
        ]
        raise RuntimeError(
            'R6 selected-component smoke checks failed: ' + ', '.join(failed)
        )
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r6_last_checkpoint_only',
        'strict_best_checkpoint_policy': (
            'save every strict numeric pilot-score improvement'
        ),
        'minimum_delta_policy': (
            'used only to reset early-stopping patience'
        ),
        'r6_preflight_gate_sha256': file_sha256(R6_PREFLIGHT_GATE_PATH),
        'r6_proposal_loss_policy_sha256': file_sha256(R6_POLICY_PATH),
        'selected_component_loss_coefficient': (
            R6_COMPONENT_LOSS_COEFFICIENT
        ),
        'selected_component_exact_coverage_weight': (
            R6_EXACT_COVERAGE_WEIGHT
        ),
        'selected_component_tolerant_coverage_weight': (
            R6_TOLERANT_COVERAGE_WEIGHT
        ),
        'selected_component_tolerance_radius_mm': (
            R6_TOLERANCE_RADIUS_MM
        ),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'r6_selected_component_smoke_checks': r6_smoke_checks,
        'r6_smoke_subject_id': smoke_subject_id,
        'r6_smoke_schedule_index': smoke_index,
        'r6_smoke_dropout_flags': smoke_dropout_flags,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R6 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R6 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r6 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R6 COMPONENT-PRESERVING FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('Selected-component proposal loss:     enabled (fit positives only)')
    print('Best-checkpoint minimum delta:        disabled (strict numeric best)')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        strict_best_improved = current_score > best_score
        patience_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if strict_best_improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
        if patience_improved:
            patience_reference_score = current_score
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        epoch_tier_metrics = {
            str(row['component_tier']): row for row in tier_rows
        }
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'small_component_proposal_within_2mm_fraction': summary[
                'small_component_proposal_within_2mm_fraction'
            ],
            'small_component_proposal_within_4mm_fraction': summary[
                'small_component_proposal_within_4mm_fraction'
            ],
            'small_component_proposal_within_8mm_fraction': summary[
                'small_component_proposal_within_8mm_fraction'
            ],
            'small_component_representation_failure_fraction': summary[
                'small_component_representation_failure_fraction'
            ],
            **{
                f'{tier_name}_{metric_name}': float(
                    epoch_tier_metrics.get(tier_name, {}).get(
                        metric_name, float('nan')
                    )
                )
                for tier_name in COMPONENT_TIERS
                for metric_name in (
                    'final_component_recall',
                    'proposal_component_recall',
                    'proposal_within_8mm_fraction',
                    'representation_failure_fraction',
                )
            },
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'strict_best_improved': strict_best_improved,
            'patience_improved_by_minimum_delta': patience_improved,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if strict_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'component={running["selected_component_coverage_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        print(
            '  R6 proposal geometry: small within 2/4/8 mm='
            f'{summary["small_component_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_8mm_fraction"]:.4f}, '
            'small representation failure='
            f'{summary["small_component_representation_failure_fraction"]:.4f}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_baseline = r6_policy['r5_baseline']
    success_policy = r6_policy['r6_directional_success_gates']
    best_tier_by_name = {
        str(row['component_tier']): row for row in best_tier_rows
    }
    best_large_proposal_recall = float(
        best_tier_by_name.get('large', {}).get(
            'proposal_component_recall', 0.0
        )
    )
    mean_dice_improvement = (
        float(best_summary['positive_mean_dice'])
        - float(r5_baseline['best_mean_dice_fixed_threshold'])
    )
    small_within_8mm_improvement = (
        float(best_summary[
            'small_component_proposal_within_8mm_fraction'
        ])
        - float(r5_baseline[
            'small_component_proposal_within_8mm_fraction'
        ])
    )
    small_representation_failure_reduction = (
        float(r5_baseline[
            'small_component_representation_failure_fraction'
        ])
        - float(best_summary[
            'small_component_representation_failure_fraction'
        ])
    )
    global_dice_regression = (
        float(r5_baseline['best_global_dice_fixed_threshold'])
        - float(best_summary['positive_global_dice'])
    )
    r6_success_checks = {
        'mean_dice_improved_by_required_margin': (
            mean_dice_improvement
            >= float(success_policy[
                'mean_dice_absolute_improvement_at_least'
            ])
        ),
        'small_component_within_8mm_improved_by_required_margin': (
            small_within_8mm_improvement
            >= float(success_policy[
                'small_component_within_8mm_absolute_improvement_at_least'
            ])
        ),
        'small_representation_failure_reduced_by_required_margin': (
            small_representation_failure_reduction
            >= float(success_policy[
                'small_representation_failure_absolute_reduction_at_least'
            ])
        ),
        'proposal_component_recall_meets_minimum': (
            float(best_summary['proposal_component_recall'])
            >= float(success_policy['proposal_component_recall_minimum'])
        ),
        'global_dice_regression_within_limit': (
            global_dice_regression
            <= float(success_policy[
                'global_dice_maximum_allowed_regression'
            ])
        ),
        'empty_false_positive_volume_within_limit': (
            float(best_summary['empty_false_positive_volume_ml'])
            <= float(success_policy[
                'mean_false_positive_volume_ml_maximum'
            ])
        ),
        'large_component_proposal_recall_meets_minimum': (
            best_large_proposal_recall
            >= float(success_policy[
                'large_component_proposal_recall_minimum'
            ])
        ),
    }
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'r5_baseline_evidence': r5_baseline,
        'r6_directional_success_policy': success_policy,
        'r6_directional_success_checks': r6_success_checks,
        'r6_directional_success_passed': all(r6_success_checks.values()),
        'r6_directional_effects': {
            'mean_dice_absolute_change': mean_dice_improvement,
            'small_component_within_8mm_absolute_change': (
                small_within_8mm_improvement
            ),
            'small_representation_failure_absolute_reduction': (
                small_representation_failure_reduction
            ),
            'global_dice_regression': global_dice_regression,
            'large_component_proposal_recall': (
                best_large_proposal_recall
            ),
        },
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R6 COMPONENT-PRESERVING FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(
        'Small proposal within 8 mm:       '
        f'{best_summary["small_component_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Small representation failure:     '
        f'{best_summary["small_component_representation_failure_fraction"]:.4f}'
    )
    print(
        'R6 directional success gates:     '
        f'{all(r6_success_checks.values())}'
    )
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


run_v22_fold0_pilot()


In [ ]:
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import ndimage


R7_PREFLIGHT_VERSION = 'v1_delayed_small_component_local_contrastive'
R7_PREFLIGHT_CODE_REVISION = 'r1_precision_bounded_diffuse_shortcut_guard'
EXPECTED_R5_PILOT_VERSION = (
    'v2_2_fold0_global_local_component_diagnostic_r5'
)
EXPECTED_R5_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
EXPECTED_R5_FUSION_DIAGNOSTIC_VERSION = (
    'v1_all20_best_vs_last_fixed_threshold'
)
EXPECTED_R5_FUSION_DIAGNOSTIC_REVISION = (
    'r3_positive_read_only_gate_assertions'
)

R7_COMPONENT_LOSS_MAX_COEFFICIENT = 0.10
R7_COMPONENT_LOSS_START_EPOCH = 8
R7_COMPONENT_LOSS_FULL_EPOCH = 12
R7_POSITIVE_WEIGHT = 0.40
R7_HARD_NEGATIVE_WEIGHT = 0.40
R7_RANK_WEIGHT = 0.20
R7_INNER_RADIUS_MM = 4.0
R7_OUTER_RADIUS_MM = 12.0
R7_HARD_NEGATIVE_RATIO = 4
R7_HARD_NEGATIVE_MINIMUM = 16
R7_LOGIT_MARGIN = 1.0
R7_VOLUME_BUDGET_MULTIPLIER = 3.0
R7_SMALL_COMPONENT_TIERS = {
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small',
}
R7_NATIVE_SPACING_MM = (0.8, 0.8, 2.0)
R7_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip(R7_NATIVE_SPACING_MM, (2, 2, 2))
)
R7_SYNTHETIC_COMPONENT_SIZES = (1, 2, 4, 8, 32, 128)
R7_EXPECTED_ATOMIC_COMPONENTS = 872
R7_EXPECTED_ARCHITECTURE_PARAMETERS = 16_870_679
R7_MANUAL_R5_DIAGNOSTIC_REVIEW_ACCEPTED = True
EXPECTED_R6_PILOT_VERSION = 'v2_2_fold0_component_preserving_proposal_r6'
EXPECTED_R6_CODE_REVISION = (
    'r6_component_equalized_proposal_strict_best_r2_smoke'
)
EXPECTED_R5_SEED = 20261002


R7_BOOTSTRAP_REQUIRED = (
    'PROJECT_ROOT', 'R2_VERSION', 'MODEL_VERSION',
    'PILOT_VERSION', 'PILOT_CODE_REVISION', 'PILOT_ROOT', 'PILOT_SEED',
    'PILOT_RESULT_GATE_PATH', 'EXPECTED_V22_PARAMETERS',
    'GLOBAL_CANVAS_SHAPE', 'GLOBAL_INPUT_SHAPE', 'GLOBAL_DOWNSAMPLE',
    'fold0_fit_df', 'fold0_heldout_df', 'locked_validation_df',
    'sealed_test_df', 'component_df', 'component_records',
    'pilot_mask_records', 'pilot_unpack_mask',
    'pilot_build_epoch_schedule', 'pilot_soft_dice_loss',
    'pilot_focal_bce_loss', 'atomic_write_csv', 'atomic_write_json',
    'file_sha256', 'load_json', 'as_bool',
)
R7_BOOTSTRAP_MISSING = [
    name for name in R7_BOOTSTRAP_REQUIRED if name not in globals()
]
if R7_BOOTSTRAP_MISSING:
    raise RuntimeError(
        'TRIAL8 R7 COMPONENT-PROPOSAL PREFLIGHT RUNTIME BOOTSTRAP REQUIRED. '
        'Run the complete current trial8_v2_2_fold0_pilot_training.py block '
        'once in this Colab runtime. A completed r5 run should print '
        '"TRIAL8 V2.2 FOLD-0 PILOT ALREADY COMPLETE" and return without '
        'retraining or rewriting model checkpoints. Then rerun this r7 '
        'preflight. Do not delete or rename the r5 directory. Missing runtime '
        'symbols: ' + ', '.join(R7_BOOTSTRAP_MISSING)
    )


R5_FUSION_GATE_PATH = (
    PILOT_ROOT / 'diagnostics' / 'fusion_attribution'
    / EXPECTED_R5_FUSION_DIAGNOSTIC_VERSION
    / 'fusion_attribution_gate.json'
)
R6_RESULT_GATE_PATH = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / EXPECTED_R6_PILOT_VERSION / 'pilot_result_gate.json'
)
R7_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r7_local_contrastive_component_proposal'
    / R7_PREFLIGHT_VERSION
)
R7_COMPONENT_AUDIT_PATH = R7_PREFLIGHT_ROOT / 'component_pooling_audit.csv'
R7_TARGET_ACCESS_AUDIT_PATH = R7_PREFLIGHT_ROOT / 'target_access_audit.csv'
R7_SCHEDULE_AUDIT_PATH = R7_PREFLIGHT_ROOT / 'schedule_component_audit.csv'
R7_SYNTHETIC_GRADIENT_PATH = (
    R7_PREFLIGHT_ROOT / 'synthetic_gradient_allocation_audit.csv'
)
R7_REAL_GRADIENT_PATH = (
    R7_PREFLIGHT_ROOT / 'real_component_gradient_audit.csv'
)
R7_TIER_SUMMARY_PATH = R7_PREFLIGHT_ROOT / 'component_tier_summary.csv'
R7_COEFFICIENT_AUDIT_PATH = (
    R7_PREFLIGHT_ROOT / 'delayed_coefficient_schedule.csv'
)
R7_SHORTCUT_AUDIT_PATH = (
    R7_PREFLIGHT_ROOT / 'diffuse_shortcut_audit.csv'
)
R7_POLICY_PATH = R7_PREFLIGHT_ROOT / 'r7_local_contrastive_policy.json'
R7_PREFLIGHT_GATE_PATH = (
    R7_PREFLIGHT_ROOT / 'r7_local_contrastive_preflight_gate.json'
)


def r7_require_runtime_contract():
    missing_paths = [
        str(path) for path in (
            PILOT_RESULT_GATE_PATH, R5_FUSION_GATE_PATH, R6_RESULT_GATE_PATH,
        )
        if not Path(path).exists()
    ]
    if missing_paths:
        raise FileNotFoundError(
            'TRIAL8 R7 PREFLIGHT PREREQUISITES ARE MISSING. Complete the r5 '
            'fold-0 pilot, its fusion diagnostic, and the r6 pilot first. '
            'Missing: ' + ', '.join(missing_paths)
        )
    r5_result = load_json(PILOT_RESULT_GATE_PATH)
    fusion_gate = load_json(R5_FUSION_GATE_PATH)
    r6_result = load_json(R6_RESULT_GATE_PATH)
    fusion_checks = fusion_gate.get('checks', {})
    evidence = fusion_gate.get('evidence_summary', {})
    checks = {
        'active_runtime_is_r5': PILOT_VERSION == EXPECTED_R5_PILOT_VERSION,
        'active_runtime_revision_matches_r5': (
            PILOT_CODE_REVISION == EXPECTED_R5_CODE_REVISION
        ),
        'r5_result_gate_exists': Path(PILOT_RESULT_GATE_PATH).exists(),
        'r5_pilot_completed': as_bool(r5_result.get('pilot_complete', False)),
        'r5_training_was_fold0_only': not as_bool(
            r5_result.get('full_fivefold_training_authorized', False)
        ),
        'fusion_gate_exists': R5_FUSION_GATE_PATH.exists(),
        'fusion_diagnostic_complete': as_bool(
            fusion_gate.get('diagnostic_complete', False)
        ),
        'fusion_diagnostic_version_matches': (
            fusion_gate.get('diagnostic_version')
            == EXPECTED_R5_FUSION_DIAGNOSTIC_VERSION
        ),
        'fusion_diagnostic_revision_matches': (
            fusion_gate.get('code_revision')
            == EXPECTED_R5_FUSION_DIAGNOSTIC_REVISION
        ),
        'all_fusion_diagnostic_checks_passed': (
            len(fusion_checks) > 0
            and all(bool(value) for value in fusion_checks.values())
        ),
        'diagnosed_bottleneck_matches_r7_intervention': (
            evidence.get('primary_evidence_class')
            == 'proposal_representation_or_supervision_bottleneck'
        ),
        'r6_pilot_version_matches': (
            r6_result.get('pilot_version') == EXPECTED_R6_PILOT_VERSION
        ),
        'r6_code_revision_matches': (
            r6_result.get('code_revision') == EXPECTED_R6_CODE_REVISION
        ),
        'r6_pilot_completed': as_bool(r6_result.get('pilot_complete', False)),
        'r6_directional_success_failed': not as_bool(
            r6_result.get('r6_directional_success_passed', True)
        ),
        'active_runtime_uses_r5_seed': int(PILOT_SEED) == EXPECTED_R5_SEED,
        'checkpoint_policy_requires_decoupling': as_bool(
            evidence.get(
                'checkpoint_selection_should_be_decoupled_from_patience',
                False,
            )
        ),
        'fold0_fit_has_82_subjects': len(fold0_fit_df) == 82,
        'fold0_heldout_has_20_subjects': len(fold0_heldout_df) == 20,
        'locked_validation_has_23_subjects': len(locked_validation_df) == 23,
        'sealed_test_has_23_subjects': len(sealed_test_df) == 23,
        'global_downsample_matches_r7_contract': (
            tuple(int(v) for v in GLOBAL_DOWNSAMPLE) == (2, 2, 2)
        ),
        'global_shape_matches_r7_contract': (
            tuple(int(v) for v in GLOBAL_INPUT_SHAPE) == (144, 160, 64)
        ),
        'architecture_parameter_count_is_unchanged': (
            int(EXPECTED_V22_PARAMETERS)
            == R7_EXPECTED_ARCHITECTURE_PARAMETERS
        ),
        'manual_r5_diagnostic_review_accepted': (
            R7_MANUAL_R5_DIAGNOSTIC_REVIEW_ACCEPTED
        ),
        'r7_output_is_outside_r5_checkpoint_tree': not str(
            R7_PREFLIGHT_ROOT
        ).startswith(str(PILOT_ROOT)),
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R7 UPSTREAM CONTRACT FAILED. Do not continue. Failed: '
            + ', '.join(failed)
        )
    return r5_result, fusion_gate, r6_result, checks


def r7_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


def r7_unique_linear_coordinates(coordinates, pad_before):
    coordinates = np.asarray(coordinates, dtype=np.int64)
    pad_before = np.asarray(pad_before, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pooled = (coordinates + pad_before[None]) // downsample[None]
    if len(pooled) == 0:
        return np.empty(0, dtype=np.int64)
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        if int(pooled[:, axis].min()) < 0 or int(pooled[:, axis].max()) >= size:
            raise RuntimeError('Pooled component coordinate is out of bounds.')
    linear = np.ravel_multi_index(pooled.T, GLOBAL_INPUT_SHAPE)
    return np.unique(linear.astype(np.int64))


def r7_dense_mask(linear_coordinates, shape=GLOBAL_INPUT_SHAPE):
    mask = np.zeros(int(np.prod(shape)), dtype=bool)
    mask[np.asarray(linear_coordinates, dtype=np.int64)] = True
    return mask.reshape(shape)


def r7_sparse_max_pool(values, pad_before):
    values = np.asarray(values, dtype=np.float32)
    coordinates = np.argwhere(values > 0)
    if len(coordinates) == 0:
        return (
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.float32),
        )
    pad_before = np.asarray(pad_before, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pooled = (coordinates + pad_before[None]) // downsample[None]
    linear = np.ravel_multi_index(pooled.T, GLOBAL_INPUT_SHAPE)
    unique, inverse = np.unique(linear.astype(np.int64), return_inverse=True)
    maxima = np.zeros(len(unique), dtype=np.float32)
    np.maximum.at(
        maxima, inverse,
        values[tuple(coordinates[:, axis] for axis in range(3))],
    )
    return unique, maxima


def r7_dense_values(linear_coordinates, values, shape=GLOBAL_INPUT_SHAPE):
    dense = np.zeros(int(np.prod(shape)), dtype=np.float32)
    dense[np.asarray(linear_coordinates, dtype=np.int64)] = np.asarray(
        values, dtype=np.float32
    )
    return dense.reshape(shape)


def r7_audit_components():
    connectivity = np.ones((3, 3, 3), dtype=bool)
    expected = component_df.copy()
    expected['subject_id'] = expected['subject_id'].astype(str)
    expected_by_subject = {
        subject_id: group.sort_values('atomic_component_id')
        for subject_id, group in expected.groupby('subject_id')
    }
    component_pool = {}
    subject_target_sparse = {}
    rows = []
    target_access_rows = []
    fit_rows = fold0_fit_df.copy()
    fit_rows['subject_id'] = fit_rows['subject_id'].astype(str)
    fit_subjects = set(fit_rows['subject_id'])
    for row in fit_rows.sort_values('subject_id').to_dict('records'):
        subject_id = str(row['subject_id'])
        npz_path = Path(str(row['vascufate_v2_npz_path']))
        with np.load(npz_path, allow_pickle=False) as data:
            required = {'target_binary', 'target_soft', 'target_spacing'}
            missing = sorted(required - set(data.files))
            if missing:
                raise RuntimeError(
                    f'{subject_id}: target arrays missing {missing}.'
                )
            binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
            soft = np.asarray(data['target_soft'], dtype=np.float32)
            spacing = np.asarray(
                data['target_spacing'], dtype=np.float64
            ).reshape(-1)
        if binary.shape != soft.shape:
            raise RuntimeError(f'{subject_id}: binary/soft shapes disagree.')
        if (
            spacing.shape != (3,)
            or not np.isfinite(spacing).all()
            or bool((spacing <= 0).any())
        ):
            raise RuntimeError(f'{subject_id}: target spacing is invalid.')
        if (
            not np.isfinite(soft).all()
            or bool((soft < 0).any())
            or bool((soft > 1).any())
        ):
            raise RuntimeError(f'{subject_id}: soft target is invalid.')

        supervision_mode = str(row['trial8_supervision_mode'])
        if supervision_mode == 'hard_and_soft':
            support = binary
        elif supervision_mode == 'soft_only_subvoxel':
            support = soft > 0
        elif supervision_mode == 'true_empty':
            support = np.zeros_like(binary, dtype=bool)
        else:
            raise RuntimeError(
                f'{subject_id}: unknown supervision mode '
                f'{supervision_mode!r}.'
            )
        cached_union = pilot_unpack_mask(
            pilot_mask_records[subject_id]['support']
        ).astype(bool)
        raw_union = binary | (soft > 0)
        if not np.array_equal(cached_union, raw_union):
            raise RuntimeError(
                f'{subject_id}: r5 cached target union does not match source.'
            )
        labels, count = ndimage.label(support, structure=connectivity)
        expected_subject = expected_by_subject.get(
            subject_id, pd.DataFrame(columns=expected.columns)
        )
        if count != len(expected_subject):
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {len(expected_subject)}.'
            )
        pad_total = (
            np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64)
            - np.asarray(support.shape, dtype=np.int64)
        )
        if bool((pad_total < 0).any()):
            raise RuntimeError(f'{subject_id}: support exceeds global canvas.')
        pad_before = pad_total // 2
        full_target = np.maximum(binary.astype(np.float32), soft)
        full_linear, full_values = r7_sparse_max_pool(
            full_target, pad_before
        )
        subject_target_sparse[subject_id] = (full_linear, full_values)
        target_access_rows.append({
            'subject_id': subject_id,
            'development_partition': 'fold0_fit',
            'supervision_mode': supervision_mode,
            'npz_path': str(npz_path),
            'binary_voxels': int(binary.sum()),
            'soft_support_voxels': int((soft > 0).sum()),
            'frozen_component_support_voxels': int(support.sum()),
            'global_target_nonzero_voxels': int(len(full_linear)),
            'spacing_x_mm': float(spacing[0]),
            'spacing_y_mm': float(spacing[1]),
            'spacing_z_mm': float(spacing[2]),
            'target_arrays_read': True,
            'fold0_heldout_target_read': False,
            'locked_validation_target_read': False,
            'test_target_read': False,
        })
        for component_id in range(1, count + 1):
            frozen = expected_subject[
                expected_subject['atomic_component_id'].astype(int)
                == component_id
            ]
            if len(frozen) != 1:
                raise RuntimeError(
                    f'{subject_id}: missing or duplicate component '
                    f'{component_id} in frozen audit.'
                )
            frozen = frozen.iloc[0]
            coordinates = np.argwhere(labels == component_id)
            pooled_linear = r7_unique_linear_coordinates(
                coordinates, pad_before
            )
            component_pool[(subject_id, component_id)] = pooled_linear
            native_voxels_match = (
                len(coordinates) == int(frozen['component_voxels'])
            )
            pooled_voxels_match = (
                len(pooled_linear)
                == int(frozen['global_proposal_voxels'])
            )
            volume_per_voxel_ml = (
                float(frozen['component_volume_ml'])
                / max(int(frozen['component_voxels']), 1)
            )
            rows.append({
                'subject_id': subject_id,
                'atomic_component_id': component_id,
                'component_tier': str(frozen['component_tier']),
                'native_component_voxels': int(len(coordinates)),
                'global_component_voxels': int(len(pooled_linear)),
                'component_volume_ml': float(
                    frozen['component_volume_ml']
                ),
                'volume_per_native_voxel_ml': volume_per_voxel_ml,
                'native_voxels_match_frozen_audit': native_voxels_match,
                'global_voxels_match_frozen_audit': pooled_voxels_match,
                'preserved_on_global_grid': len(pooled_linear) > 0,
                'development_partition': 'fold0_fit',
                'target_array_opened_by_r7_block': True,
            })

    audit = pd.DataFrame(rows)
    target_access = pd.DataFrame(target_access_rows)
    if len(audit):
        full_counts = {
            subject_id: len(values[0])
            for subject_id, values in subject_target_sparse.items()
        }
        audit['full_subject_global_positive_voxels'] = audit[
            'subject_id'
        ].map(full_counts).astype(int)
        audit['current_voxel_weighted_mass_fraction'] = (
            audit['global_component_voxels']
            / audit['full_subject_global_positive_voxels'].clip(lower=1)
        )
        audit['r7_selected_component_loss_units'] = 1.0
    return audit, component_pool, subject_target_sparse, target_access


def r7_audit_schedule(component_pool):
    rows = []
    heldout_subjects = set(fold0_heldout_df['subject_id'].astype(str))
    for epoch in (0, 1):
        first = pilot_build_epoch_schedule(epoch)
        second = pilot_build_epoch_schedule(epoch)
        if first != second:
            raise RuntimeError(f'Epoch {epoch} schedule is not deterministic.')
        for schedule_index, entry in enumerate(first):
            subject_id = str(entry['subject_id'])
            sample_type = str(entry['sample_type'])
            component_id = int(entry['atomic_component_id'])
            valid = sample_type == 'positive'
            small_tier = str(entry['component_tier']) in R7_SMALL_COMPONENT_TIERS
            key = (subject_id, component_id)
            mapped = key in component_pool if valid else component_id == 0
            component_voxels = (
                int(len(component_pool[key])) if valid and mapped else 0
            )
            rows.append({
                'epoch': int(epoch),
                'schedule_index': int(schedule_index),
                'sample_type': sample_type,
                'subject_id': subject_id,
                'atomic_component_id': component_id,
                'component_tier': str(entry['component_tier']),
                'component_volume_ml': float(entry['component_volume_ml']),
                'positive_component_sample': valid,
                'local_contrastive_loss_eligible': valid and small_tier,
                'selected_component_maps_to_global_grid': bool(mapped),
                'selected_global_component_voxels': component_voxels,
                'subject_is_fold0_heldout': subject_id in heldout_subjects,
            })
    return pd.DataFrame(rows)


def r7_component_coefficient(epoch):
    
    epoch = int(epoch)
    if epoch < R7_COMPONENT_LOSS_START_EPOCH:
        return 0.0
    if R7_COMPONENT_LOSS_FULL_EPOCH <= R7_COMPONENT_LOSS_START_EPOCH:
        return float(R7_COMPONENT_LOSS_MAX_COEFFICIENT)
    progress = (
        (epoch - R7_COMPONENT_LOSS_START_EPOCH + 1)
        / (R7_COMPONENT_LOSS_FULL_EPOCH - R7_COMPONENT_LOSS_START_EPOCH + 1)
    )
    return float(R7_COMPONENT_LOSS_MAX_COEFFICIENT * np.clip(progress, 0, 1))


def r7_component_regions(exact_target, full_target, valid_domain):
    exact = np.asarray(exact_target, dtype=bool)
    full = np.asarray(full_target) > 0
    domain = np.asarray(valid_domain, dtype=bool)
    inner = ndimage.binary_dilation(
        exact,
        structure=r7_physical_ball(
            R7_GLOBAL_SPACING_MM, R7_INNER_RADIUS_MM
        ),
    )
    outer = ndimage.binary_dilation(
        exact,
        structure=r7_physical_ball(
            R7_GLOBAL_SPACING_MM, R7_OUTER_RADIUS_MM
        ),
    )
    negative = outer & ~inner & domain & ~full
    return exact, inner, outer, negative


def r7_local_contrastive_loss(
    logits, exact_component_target, negative_annulus_target,
    component_valid,
):
    exact_target = exact_component_target.bool()
    negative_target = negative_annulus_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    positive_losses = []
    negative_losses = []
    rank_losses = []
    positive_logits = []
    hard_negative_logits = []
    selected_negative_counts = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]].float()
        negative = logits[batch_index][negative_target[batch_index]].float()
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError(
                'A valid r7 component must have positive voxels and a '
                'lesion-free local negative annulus.'
            )
        hard_count = min(
            int(negative.numel()),
            max(
                R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * int(positive.numel()),
            ),
        )
        hard_negative = torch.topk(
            negative, k=hard_count, largest=True
        ).values
        positive_loss = F.softplus(-positive).mean()
        negative_loss = F.softplus(hard_negative).mean()
        rank_loss = F.softplus(
            R7_LOGIT_MARGIN - positive.mean() + hard_negative.mean()
        )
        loss = (
            R7_POSITIVE_WEIGHT * positive_loss
            + R7_HARD_NEGATIVE_WEIGHT * negative_loss
            + R7_RANK_WEIGHT * rank_loss
        )
        losses.append(loss)
        positive_losses.append(positive_loss)
        negative_losses.append(negative_loss)
        rank_losses.append(rank_loss)
        positive_logits.append(positive.mean())
        hard_negative_logits.append(hard_negative.mean())
        selected_negative_counts.append(hard_count)
    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'local_contrastive_loss': zero,
            'component_positive_loss': zero,
            'component_hard_negative_loss': zero,
            'component_rank_loss': zero,
            'mean_component_positive_logit': zero.detach(),
            'mean_component_hard_negative_logit': zero.detach(),
            'selected_hard_negative_voxels': 0,
            'selected_components_in_batch': 0,
        }
    return torch.stack(losses).mean(), {
        'local_contrastive_loss': torch.stack(losses).mean(),
        'component_positive_loss': torch.stack(positive_losses).mean(),
        'component_hard_negative_loss': torch.stack(negative_losses).mean(),
        'component_rank_loss': torch.stack(rank_losses).mean(),
        'mean_component_positive_logit': torch.stack(positive_logits).mean(),
        'mean_component_hard_negative_logit': torch.stack(
            hard_negative_logits
        ).mean(),
        'selected_hard_negative_voxels': int(sum(selected_negative_counts)),
        'selected_components_in_batch': len(losses),
    }


def r7_current_proposal_loss(logits, full_target):
    dice = pilot_soft_dice_loss(logits, full_target).mean()
    focal_bce = pilot_focal_bce_loss(logits, full_target).mean()
    return 0.65 * dice + 0.35 * focal_bce, {
        'full_proposal_dice_loss': dice,
        'full_proposal_focal_bce_loss': focal_bce,
    }


def r7_component_preserving_proposal_loss(
    logits, full_target, exact_component_target,
    negative_annulus_target, component_valid, coefficient,
):
    current, current_parts = r7_current_proposal_loss(logits, full_target)
    contrastive, contrastive_parts = r7_local_contrastive_loss(
        logits, exact_component_target, negative_annulus_target,
        component_valid,
    )
    total = current + float(coefficient) * contrastive
    return total, {
        **current_parts,
        **contrastive_parts,
        'current_whole_mask_proposal_loss': current,
        'component_loss_coefficient': float(coefficient),
        'component_preserving_proposal_loss': total,
    }


def r7_gradient_record(
    logits_shape, full_target_np, exact_target_np, valid_domain_np, tier, label,
):
    full_target = torch.from_numpy(
        np.asarray(full_target_np, dtype=np.float32)[None, None]
    )
    exact_target = torch.from_numpy(
        np.asarray(exact_target_np, dtype=bool)[None, None]
    )
    _, inner_np, outer_np, negative_np = r7_component_regions(
        exact_target_np, full_target_np, valid_domain_np
    )
    negative_target = torch.from_numpy(negative_np[None, None])
    valid = torch.ones(1, dtype=torch.bool)

    old_logits = torch.full(
        logits_shape, -2.0, dtype=torch.float32, requires_grad=True
    )
    old_loss, _ = r7_current_proposal_loss(old_logits, full_target)
    old_gradient = torch.autograd.grad(old_loss, old_logits)[0].detach()

    new_logits = torch.full(
        logits_shape, -2.0, dtype=torch.float32, requires_grad=True
    )
    new_loss, parts = r7_component_preserving_proposal_loss(
        new_logits, full_target, exact_target, negative_target, valid,
        R7_COMPONENT_LOSS_MAX_COEFFICIENT,
    )
    new_gradient = torch.autograd.grad(new_loss, new_logits)[0].detach()

    exact = exact_target.bool()
    negative = negative_target.bool()
    outer = torch.from_numpy(outer_np[None, None])
    outside_outer = ~outer
    old_selected = float(old_gradient[exact].abs().sum())
    new_selected = float(new_gradient[exact].abs().sum())
    negative_gradient = new_gradient[negative]
    positive_gradient = new_gradient[exact]
    outside_delta = (
        new_gradient[outside_outer] - old_gradient[outside_outer]
    ).abs()
    outside_difference = (
        float(outside_delta.max()) if outside_delta.numel() else 0.0
    )
    record = {
        'probe': str(label),
        'component_tier': str(tier),
        'component_global_voxels': int(exact.sum()),
        'inner_global_voxels': int(inner_np.sum()),
        'outer_global_voxels': int(outer_np.sum()),
        'negative_annulus_voxels': int(negative.sum()),
        'current_loss': float(old_loss.detach()),
        'r7_loss': float(new_loss.detach()),
        'local_contrastive_loss': float(
            parts['local_contrastive_loss'].detach()
        ),
        'current_selected_gradient_l1': old_selected,
        'r7_selected_gradient_l1': new_selected,
        'selected_gradient_amplification': (
            new_selected / max(old_selected, 1e-12)
        ),
        'current_total_gradient_l1': float(old_gradient.abs().sum()),
        'r7_total_gradient_l1': float(new_gradient.abs().sum()),
        'selected_positive_gradient_is_descent_up': bool(
            positive_gradient.numel() > 0
            and float(positive_gradient.max()) < 0.0
        ),
        'annulus_gradient_is_descent_down': bool(
            negative_gradient.numel() > 0
            and float(negative_gradient.min()) > 0.0
        ),
        'maximum_gradient_change_outside_outer_region': outside_difference,
        'all_gradients_finite': bool(
            torch.isfinite(old_gradient).all()
            and torch.isfinite(new_gradient).all()
        ),
    }
    del old_logits, new_logits, old_gradient, new_gradient
    gc.collect()
    return record


def r7_synthetic_gradient_audit():
    shape = (32, 32, 16)
    large = np.zeros(shape, dtype=bool)
    large[2:10, 2:10, 2:6] = True
    candidate = np.argwhere(
        np.pad(
            np.ones((10, 10, 7), dtype=bool),
            ((20, 2), (20, 2), (8, 1)),
            mode='constant', constant_values=False,
        )
    )
    rows = []
    for size in R7_SYNTHETIC_COMPONENT_SIZES:
        exact = np.zeros(shape, dtype=bool)
        coordinates = candidate[:size]
        exact[tuple(coordinates[:, axis] for axis in range(3))] = True
        full = large | exact
        rows.append(r7_gradient_record(
            (1, 1, *shape), full, exact, np.ones(shape, dtype=bool),
            tier=f'synthetic_{size}_voxels', label=f'synthetic_{size}',
        ))
    return pd.DataFrame(rows)


def r7_coefficient_audit():
    return pd.DataFrame([
        {
            'epoch': epoch,
            'component_loss_coefficient': r7_component_coefficient(epoch),
            'r5_proposal_objective_is_exactly_preserved': (
                r7_component_coefficient(epoch) == 0.0
            ),
        }
        for epoch in range(1, 41)
    ])


def r7_diffuse_shortcut_audit():
    shape = (32, 32, 16)
    exact_np = np.zeros(shape, dtype=bool)
    exact_np[16, 16, 8] = True
    full_np = exact_np.astype(np.float32)
    domain_np = np.ones(shape, dtype=bool)
    _, _, _, negative_np = r7_component_regions(
        exact_np, full_np, domain_np
    )
    exact = torch.from_numpy(exact_np[None, None])
    negative = torch.from_numpy(negative_np[None, None])
    valid = torch.ones(1, dtype=torch.bool)

    rows = []
    profiles = {
        'diffuse_high': np.full(shape, 4.0, dtype=np.float32),
        'localized_high': np.full(shape, -4.0, dtype=np.float32),
    }
    profiles['localized_high'][exact_np] = 4.0
    for name, values in profiles.items():
        logits = torch.tensor(
            values[None, None], dtype=torch.float32, requires_grad=True
        )
        loss, parts = r7_local_contrastive_loss(
            logits, exact, negative, valid
        )
        gradient = torch.autograd.grad(loss, logits)[0].detach()
        rows.append({
            'profile': name,
            'local_contrastive_loss': float(loss.detach()),
            'positive_probability': float(
                torch.sigmoid(logits.detach()[exact]).mean()
            ),
            'annulus_probability': float(
                torch.sigmoid(logits.detach()[negative]).mean()
            ),
            'positive_gradient_mean': float(gradient[exact].mean()),
            'annulus_gradient_mean': float(gradient[negative].mean()),
            'positive_minus_hard_negative_logit': float(
                parts['mean_component_positive_logit'].detach()
                - parts['mean_component_hard_negative_logit'].detach()
            ),
        })

    trainable = torch.zeros(
        (1, 1, *shape), dtype=torch.float32, requires_grad=True
    )
    before, _ = r7_local_contrastive_loss(
        trainable, exact, negative, valid
    )
    gradient = torch.autograd.grad(before, trainable)[0]
    with torch.no_grad():
        updated = trainable - 8.0 * gradient
    after, after_parts = r7_local_contrastive_loss(
        updated, exact, negative, valid
    )
    rows.append({
        'profile': 'one_gradient_step',
        'local_contrastive_loss': float(after.detach()),
        'positive_probability': float(torch.sigmoid(updated[exact]).mean()),
        'annulus_probability': float(torch.sigmoid(updated[negative]).mean()),
        'positive_gradient_mean': float(gradient[exact].mean()),
        'annulus_gradient_mean': float(gradient[negative].mean()),
        'positive_minus_hard_negative_logit': float(
            after_parts['mean_component_positive_logit'].detach()
            - after_parts['mean_component_hard_negative_logit'].detach()
        ),
        'loss_before_step': float(before.detach()),
    })
    return pd.DataFrame(rows)


def r7_representative_components(component_audit):
    rows = []
    for tier, group in component_audit.groupby('component_tier'):
        ordered = group.sort_values([
            'global_component_voxels', 'subject_id', 'atomic_component_id'
        ]).reset_index(drop=True)
        rows.append(ordered.iloc[len(ordered) // 2])
    return pd.DataFrame(rows).reset_index(drop=True)


def r7_real_gradient_audit(
    component_audit, component_pool, subject_target_sparse,
):
    representatives = r7_representative_components(component_audit)
    rows = []
    for record in representatives.to_dict('records'):
        subject_id = str(record['subject_id'])
        component_id = int(record['atomic_component_id'])
        full_linear, full_values = subject_target_sparse[subject_id]
        full = r7_dense_values(full_linear, full_values)
        exact = r7_dense_mask(component_pool[(subject_id, component_id)])
        native_shape = np.asarray(
            pilot_mask_records[subject_id]['shape'], dtype=np.int64
        )
        pad_total = (
            np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64) - native_shape
        )
        pad_before = pad_total // 2
        pad_after = pad_total - pad_before
        native_domain = np.ones(tuple(native_shape), dtype=np.uint8)
        padded_domain = np.pad(
            native_domain,
            tuple((int(a), int(b)) for a, b in zip(pad_before, pad_after)),
            mode='constant', constant_values=0,
        )
        domain = padded_domain[
            ::int(GLOBAL_DOWNSAMPLE[0]),
            ::int(GLOBAL_DOWNSAMPLE[1]),
            ::int(GLOBAL_DOWNSAMPLE[2]),
        ].astype(bool)
        if domain.shape != tuple(GLOBAL_INPUT_SHAPE):
            raise RuntimeError(f'{subject_id}: invalid pooled domain shape.')
        row = r7_gradient_record(
            (1, 1, *GLOBAL_INPUT_SHAPE), full, exact, domain,
            tier=str(record['component_tier']),
            label=f'{subject_id}_component_{component_id}',
        )
        row.update({
            'subject_id': subject_id,
            'atomic_component_id': component_id,
            'component_volume_ml': float(record['component_volume_ml']),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def r7_tier_summary(component_audit, schedule_audit):
    component_summary = component_audit.groupby(
        'component_tier', as_index=False
    ).agg(
        components=('atomic_component_id', 'count'),
        subjects=('subject_id', 'nunique'),
        minimum_global_voxels=('global_component_voxels', 'min'),
        median_global_voxels=('global_component_voxels', 'median'),
        maximum_global_voxels=('global_component_voxels', 'max'),
        median_current_voxel_mass_fraction=(
            'current_voxel_weighted_mass_fraction', 'median'
        ),
        maximum_current_voxel_mass_fraction=(
            'current_voxel_weighted_mass_fraction', 'max'
        ),
    )
    selected = schedule_audit[
        schedule_audit['local_contrastive_loss_eligible']
    ].groupby('component_tier').size().rename(
        'two_epoch_contrastive_selections'
    )
    return component_summary.merge(
        selected, left_on='component_tier', right_index=True, how='left'
    ).fillna({'two_epoch_contrastive_selections': 0})


def r7_display(frame):
    if 'display' in globals():
        display(frame)
    else:
        print(frame.to_string(index=False))


def run_r7_component_proposal_preflight():
    start_time = time.time()
    r5_result, fusion_gate, r6_result, runtime_checks = (
        r7_require_runtime_contract()
    )
    if R7_PREFLIGHT_GATE_PATH.exists():
        completed = load_json(R7_PREFLIGHT_GATE_PATH)
        if (
            completed.get('preflight_version') == R7_PREFLIGHT_VERSION
            and completed.get('code_revision') == R7_PREFLIGHT_CODE_REVISION
            and as_bool(completed.get('preflight_passed', False))
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {R7_PREFLIGHT_GATE_PATH}')
            print('No r7 artifact was rewritten.')
            return
        raise RuntimeError(
            'An incompatible or incomplete artifact already occupies the r7 '
            'preflight directory. Refusing to overwrite it.'
        )
    required_paths = [
        Path(PILOT_RESULT_GATE_PATH), R5_FUSION_GATE_PATH,
        R6_RESULT_GATE_PATH,
    ]
    source_hashes_before = {
        str(path): file_sha256(path) for path in required_paths
    }

    (
        component_audit, component_pool,
        subject_target_sparse, target_access_audit,
    ) = r7_audit_components()
    schedule_audit = r7_audit_schedule(component_pool)
    synthetic_gradient = r7_synthetic_gradient_audit()
    real_gradient = r7_real_gradient_audit(
        component_audit, component_pool, subject_target_sparse
    )
    coefficient_audit = r7_coefficient_audit()
    shortcut_audit = r7_diffuse_shortcut_audit()
    tier_summary = r7_tier_summary(component_audit, schedule_audit)

    source_hashes_after = {
        str(path): file_sha256(path) for path in required_paths
    }
    positive_schedule = schedule_audit[
        schedule_audit['positive_component_sample']
    ]
    negative_schedule = schedule_audit[
        ~schedule_audit['positive_component_sample']
    ]
    eligible_schedule = schedule_audit[
        schedule_audit['local_contrastive_loss_eligible']
    ]
    expected_voxel_volume_ml = float(
        np.prod(R7_NATIVE_SPACING_MM) / 1000.0
    )
    synthetic_ratio_current = float(
        synthetic_gradient['current_selected_gradient_l1'].max()
        / synthetic_gradient['current_selected_gradient_l1'].min()
    )
    synthetic_ratio_r7 = float(
        synthetic_gradient['r7_selected_gradient_l1'].max()
        / synthetic_gradient['r7_selected_gradient_l1'].min()
    )
    small_real = real_gradient[
        real_gradient['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    diffuse = shortcut_audit.set_index('profile').loc['diffuse_high']
    localized = shortcut_audit.set_index('profile').loc['localized_high']
    one_step = shortcut_audit.set_index('profile').loc['one_gradient_step']

    checks = {
        **runtime_checks,
        'component_audit_has_872_atomic_components': (
            len(component_audit) == R7_EXPECTED_ATOMIC_COMPONENTS
        ),
        'all_component_subjects_are_fold0_fit': set(
            component_audit['subject_id']
        ).issubset(set(fold0_fit_df['subject_id'].astype(str))),
        'native_component_counts_match_frozen_audit': bool(
            component_audit[
                'native_voxels_match_frozen_audit'
            ].all()
        ),
        'global_component_counts_match_frozen_audit': bool(
            component_audit[
                'global_voxels_match_frozen_audit'
            ].all()
        ),
        'all_components_preserved_on_global_grid': bool(
            component_audit['preserved_on_global_grid'].all()
        ),
        'fixed_spacing_matches_component_volume_metadata': bool(
            np.allclose(
                component_audit['volume_per_native_voxel_ml'],
                expected_voxel_volume_ml, rtol=0.0, atol=1e-7,
            )
        ),
        'source_target_spacing_matches_r7_contract': bool(np.allclose(
            target_access_audit[[
                'spacing_x_mm', 'spacing_y_mm', 'spacing_z_mm'
            ]].to_numpy(dtype=np.float64),
            np.asarray(R7_NATIVE_SPACING_MM, dtype=np.float64)[None],
            rtol=0.0, atol=1e-6,
        )),
        'two_epoch_schedule_has_336_samples': len(schedule_audit) == 336,
        'two_epoch_schedule_has_224_positive_samples': (
            len(positive_schedule) == 224
        ),
        'two_epoch_schedule_has_112_negative_samples': (
            len(negative_schedule) == 112
        ),
        'all_positive_schedule_components_map_to_global_grid': bool(
            positive_schedule[
                'selected_component_maps_to_global_grid'
            ].all()
            and (positive_schedule[
                'selected_global_component_voxels'
            ] > 0).all()
        ),
        'negative_samples_do_not_receive_component_loss': bool(
            (~negative_schedule['local_contrastive_loss_eligible']).all()
            and (negative_schedule[
                'selected_global_component_voxels'
            ] == 0).all()
        ),
        'heldout_subjects_not_sampled': bool(
            (~schedule_audit['subject_is_fold0_heldout']).all()
        ),
        'contrastive_supervision_is_limited_to_small_tiers': (
            set(eligible_schedule['component_tier'])
            == R7_SMALL_COMPONENT_TIERS
        ),
        'medium_and_large_components_use_whole_mask_loss_only': bool((
            ~positive_schedule[
                positive_schedule['component_tier'].isin({'medium', 'large'})
            ]['local_contrastive_loss_eligible']
        ).all()),
        'epochs_1_through_7_exactly_preserve_r5_objective': bool((
            coefficient_audit.loc[
                coefficient_audit['epoch'].between(1, 7),
                'component_loss_coefficient',
            ] == 0.0
        ).all()),
        'component_loss_begins_at_epoch_8': (
            float(coefficient_audit.loc[
                coefficient_audit['epoch'] == 8,
                'component_loss_coefficient',
            ].iloc[0]) > 0.0
        ),
        'component_loss_reaches_only_frozen_low_maximum': bool(np.isclose(
            coefficient_audit['component_loss_coefficient'].max(),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT,
        )),
        'synthetic_gradients_are_finite': bool(
            synthetic_gradient['all_gradients_finite'].all()
        ),
        'real_gradients_are_finite': bool(
            real_gradient['all_gradients_finite'].all()
        ),
        'r7_equalizes_synthetic_component_gradient_mass': (
            synthetic_ratio_r7 < synthetic_ratio_current
            and synthetic_ratio_r7 <= 2.0
        ),
        'r7_amplifies_every_small_tier_probe': bool(
            len(small_real) == 4
            and (small_real['selected_gradient_amplification'] > 1.0).all()
        ),
        'small_tier_positive_gradients_raise_component_logits': bool(
            small_real['selected_positive_gradient_is_descent_up'].all()
        ),
        'small_tier_annulus_gradients_lower_false_positive_logits': bool(
            small_real['annulus_gradient_is_descent_down'].all()
        ),
        'whole_mask_gradient_is_unchanged_outside_12mm_region': bool(
            (synthetic_gradient[
                'maximum_gradient_change_outside_outer_region'
            ] <= 1e-12).all()
            and (real_gradient[
                'maximum_gradient_change_outside_outer_region'
            ] <= 1e-12).all()
        ),
        'diffuse_high_map_is_penalized_more_than_localized_map': (
            float(diffuse['local_contrastive_loss'])
            > float(localized['local_contrastive_loss']) + 1.0
        ),
        'diffuse_map_receives_downward_annulus_gradient': (
            float(diffuse['annulus_gradient_mean']) > 0.0
        ),
        'one_gradient_step_increases_local_logit_margin': (
            float(one_step['positive_minus_hard_negative_logit']) > 0.0
        ),
        'one_gradient_step_reduces_contrastive_loss': (
            float(one_step['local_contrastive_loss'])
            < float(one_step['loss_before_step'])
        ),
        'r5_and_r6_gate_hashes_unchanged': (
            source_hashes_before == source_hashes_after
        ),
        'exactly_82_fold0_fit_targets_read': (
            len(target_access_audit) == 82
            and target_access_audit['subject_id'].nunique() == 82
        ),
        'target_reads_are_exactly_fold0_fit': set(
            target_access_audit['subject_id']
        ) == set(fold0_fit_df['subject_id'].astype(str)),
        'all_component_targets_rebuilt_from_source': bool(
            component_audit['target_array_opened_by_r7_block'].all()
        ),
        'target_access_audit_reports_no_heldout_reads': bool(
            (~target_access_audit['fold0_heldout_target_read']).all()
        ),
        'target_access_audit_reports_no_locked_validation_reads': bool(
            (~target_access_audit['locked_validation_target_read']).all()
        ),
        'target_access_audit_reports_no_test_reads': bool(
            (~target_access_audit['test_target_read']).all()
        ),
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'outcome_workbooks_not_read': True,
        'study_model_not_constructed': True,
        'study_trainable_parameters_are_zero': True,
        'optimizer_was_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'r5_and_r6_artifacts_not_modified': (
            source_hashes_before == source_hashes_after
        ),
        'r7_pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        print('\nR7 preflight checks:')
        r7_display(pd.DataFrame({
            'check': list(checks), 'passed': list(checks.values())
        }))
        raise RuntimeError(
            'TRIAL8 R7 LOCAL-CONTRASTIVE PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': R7_PREFLIGHT_VERSION,
        'code_revision': R7_PREFLIGHT_CODE_REVISION,
        'parent_pilot_version': EXPECTED_R5_PILOT_VERSION,
        'architecture_parameters': int(EXPECTED_V22_PARAMETERS),
        'new_trainable_parameters': 0,
        'fresh_start_from_initialization': True,
        'whole_mask_proposal_objective_preserved': {
            'soft_dice_weight': 0.65,
            'focal_bce_weight': 0.35,
        },
        'selected_component_objective': {
            'eligible_tiers': sorted(R7_SMALL_COMPONENT_TIERS),
            'start_epoch': R7_COMPONENT_LOSS_START_EPOCH,
            'full_coefficient_epoch': R7_COMPONENT_LOSS_FULL_EPOCH,
            'maximum_coefficient': R7_COMPONENT_LOSS_MAX_COEFFICIENT,
            'positive_weight': R7_POSITIVE_WEIGHT,
            'hard_negative_weight': R7_HARD_NEGATIVE_WEIGHT,
            'rank_weight': R7_RANK_WEIGHT,
            'inner_radius_mm': R7_INNER_RADIUS_MM,
            'outer_radius_mm': R7_OUTER_RADIUS_MM,
            'hard_negative_ratio': R7_HARD_NEGATIVE_RATIO,
            'hard_negative_minimum': R7_HARD_NEGATIVE_MINIMUM,
            'logit_margin': R7_LOGIT_MARGIN,
            'negative_annulus': (
                '4-12 mm, inside final array, excluding every lesion voxel'
            ),
            'medium_and_large_policy': 'whole-mask proposal objective only',
            'negative_sample_policy': 'whole-mask proposal objective only',
        },
        'experiment_seed': EXPECTED_R5_SEED,
        'outer_training_loss_proposal_weight': 0.35,
        'checkpoint_policy': {
            'save_checkpoint_on': 'every strict numeric score improvement',
            'minimum_delta_use': 'early-stopping patience reset only',
            'minimum_delta_must_not_block_checkpoint_save': True,
        },
        'r5_baseline': fusion_gate['evidence_summary'],
        'r7_directional_success_gates': {
            'mean_dice_maximum_allowed_regression': 0.005,
            'mean_dice_target_absolute_improvement': 0.010,
            'global_dice_maximum_allowed_regression': 0.010,
            'proposal_dice_maximum_allowed_regression': 0.020,
            'volume_bounded_small_within_8mm_absolute_improvement_at_least': 0.10,
            'small_representation_failure_absolute_reduction_at_least': 0.10,
            'proposal_component_recall_minimum': 0.08,
            'mean_false_positive_volume_ml_maximum': 30.0,
            'large_component_proposal_recall_minimum': 0.83,
            'proposal_predicted_to_target_volume_ratio_maximum': 5.0,
        },
        'proposal_diagnostics': {
            'retain_raw_distance_metrics': True,
            'raw_distance_metrics_are_not_success_gates': True,
            'report_proposal_precision': True,
            'report_predicted_to_target_volume_ratio': True,
            'report_voxel_average_precision': True,
            'volume_budget_multiplier': R7_VOLUME_BUDGET_MULTIPLIER,
            'report_volume_bounded_component_recall': True,
            'report_volume_bounded_small_within_2_4_8mm': True,
        },
        'r6_failure_evidence': {
            'best_epoch': r6_result.get('best_epoch'),
            'best_score': r6_result.get('best_score'),
            'best_validation_summary': r6_result.get(
                'best_validation_summary', {}
            ),
            'directional_success_passed': r6_result.get(
                'r6_directional_success_passed', False
            ),
            'directional_effects': r6_result.get(
                'r6_directional_effects', {}
            ),
        },
        'fold0_heldout_targets_available_to_training': False,
        'locked_validation_targets_available_to_training': False,
        'test_targets_available': False,
        'r7_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'manual_r5_diagnostic_review_accepted': (
            R7_MANUAL_R5_DIAGNOSTIC_REVIEW_ACCEPTED
        ),
        'target_access_policy': {
            'authorized_split': 'development train / fold0 fit only',
            'fit_subjects_read': 82,
            'fold0_heldout_targets_read': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        },
    }

    R7_PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(component_audit, R7_COMPONENT_AUDIT_PATH)
    atomic_write_csv(target_access_audit, R7_TARGET_ACCESS_AUDIT_PATH)
    atomic_write_csv(schedule_audit, R7_SCHEDULE_AUDIT_PATH)
    atomic_write_csv(synthetic_gradient, R7_SYNTHETIC_GRADIENT_PATH)
    atomic_write_csv(real_gradient, R7_REAL_GRADIENT_PATH)
    atomic_write_csv(tier_summary, R7_TIER_SUMMARY_PATH)
    atomic_write_csv(coefficient_audit, R7_COEFFICIENT_AUDIT_PATH)
    atomic_write_csv(shortcut_audit, R7_SHORTCUT_AUDIT_PATH)
    atomic_write_json(policy, R7_POLICY_PATH)

    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'preflight_version': R7_PREFLIGHT_VERSION,
        'code_revision': R7_PREFLIGHT_CODE_REVISION,
        'preflight_passed': True,
        'checks': checks,
        'source_gate_sha256': source_hashes_before,
        'components_audited': int(len(component_audit)),
        'schedule_samples_audited': int(len(schedule_audit)),
        'synthetic_gradient_probes': int(len(synthetic_gradient)),
        'real_gradient_probes': int(len(real_gradient)),
        'diffuse_shortcut_profiles': int(len(shortcut_audit)),
        'current_synthetic_gradient_mass_ratio': synthetic_ratio_current,
        'r7_synthetic_gradient_mass_ratio': synthetic_ratio_r7,
        'architecture_parameters': int(EXPECTED_V22_PARAMETERS),
        'new_trainable_parameters': 0,
        'target_arrays_opened_by_r7_block': True,
        'target_arrays_read_splits': ['fold0_fit'],
        'target_subjects_read': int(len(target_access_audit)),
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'study_trainable_parameters': 0,
        'optimizer_steps': 0,
        'training_performed': False,
        'checkpoint_written': False,
        'r7_pilot_implementation_allowed': True,
        'r7_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'policy_path': str(R7_POLICY_PATH),
        'component_audit_path': str(R7_COMPONENT_AUDIT_PATH),
        'target_access_audit_path': str(R7_TARGET_ACCESS_AUDIT_PATH),
        'schedule_audit_path': str(R7_SCHEDULE_AUDIT_PATH),
        'synthetic_gradient_audit_path': str(R7_SYNTHETIC_GRADIENT_PATH),
        'real_gradient_audit_path': str(R7_REAL_GRADIENT_PATH),
        'tier_summary_path': str(R7_TIER_SUMMARY_PATH),
        'coefficient_audit_path': str(R7_COEFFICIENT_AUDIT_PATH),
        'diffuse_shortcut_audit_path': str(R7_SHORTCUT_AUDIT_PATH),
        'elapsed_seconds': float(time.time() - start_time),
    }
    atomic_write_json(gate, R7_PREFLIGHT_GATE_PATH)

    print('\nComponent supervision by tier:')
    r7_display(tier_summary)
    print('\nSynthetic gradient-allocation audit:')
    r7_display(synthetic_gradient)
    print('\nReal representative-component gradient audit:')
    r7_display(real_gradient)
    print('\nDelayed coefficient schedule:')
    r7_display(coefficient_audit)
    print('\nDiffuse-shortcut audit:')
    r7_display(shortcut_audit)
    print('\nR7 preflight checks:')
    r7_display(pd.DataFrame({
        'check': list(checks), 'passed': list(checks.values())
    }))

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE PROPOSAL PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Fold-0 fit subjects:                 {len(fold0_fit_df)}')
    print(f'Atomic components audited:          {len(component_audit)}')
    print(f'Two-epoch schedule samples:         {len(schedule_audit)}')
    print('Architecture parameters:            '
          f'{EXPECTED_V22_PARAMETERS:,} (unchanged)')
    print('New trainable parameters:            0')
    print('Whole-mask proposal loss:            preserved exactly')
    print('R5 seed restored:                     20261002')
    print('Contrastive-loss start epoch:         '
          f'{R7_COMPONENT_LOSS_START_EPOCH}')
    print('Maximum component coefficient:        '
          f'{R7_COMPONENT_LOSS_MAX_COEFFICIENT:.2f}')
    print('Negative annulus:                     '
          f'{R7_INNER_RADIUS_MM:.1f}-{R7_OUTER_RADIUS_MM:.1f} mm')
    print('Diffuse-shortcut guard:               passed')
    print('Volume-bounded diagnostics required:  True')
    print('Fold-0 fit targets read for audit:    82')
    print('Fold-0 held-out targets read:         False')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Training performed:                   False')
    print('R7 pilot implementation allowed:      True')
    print('R7 pilot training authorized:         False')
    print('Full five-fold training authorized:   False')
    print(f'Policy:                              {R7_POLICY_PATH}')
    print(f'Preflight gate:                      {R7_PREFLIGHT_GATE_PATH}')
    print('\nNext: build the isolated fresh-start fold-0 r7 pilot training '
          'block from this frozen loss and diagnostic contract; do not '
          'launch five folds.')


run_r7_component_proposal_preflight()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = (
    'r7_delayed_local_contrastive_volume_bounded_r1'
)
PILOT_VERSION = 'v2_2_fold0_local_contrastive_proposal_r7'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

R7_PREFLIGHT_VERSION = 'v1_delayed_small_component_local_contrastive'
R7_PREFLIGHT_CODE_REVISION = (
    'r1_precision_bounded_diffuse_shortcut_guard'
)
R7_COMPONENT_LOSS_MAX_COEFFICIENT = 0.10
R7_COMPONENT_LOSS_START_EPOCH = 8
R7_COMPONENT_LOSS_FULL_EPOCH = 12
R7_POSITIVE_WEIGHT = 0.40
R7_HARD_NEGATIVE_WEIGHT = 0.40
R7_RANK_WEIGHT = 0.20
R7_INNER_RADIUS_MM = 4.0
R7_OUTER_RADIUS_MM = 12.0
R7_HARD_NEGATIVE_RATIO = 4
R7_HARD_NEGATIVE_MINIMUM = 16
R7_LOGIT_MARGIN = 1.0
R7_VOLUME_BUDGET_MULTIPLIER = 3.0
R7_SMALL_COMPONENT_TIERS = set(COMPONENT_TIERS[:4])
R5_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
R5_PILOT_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
R7_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip((0.8, 0.8, 2.0), GLOBAL_DOWNSAMPLE)
)

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'
R7_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r7_local_contrastive_component_proposal' / R7_PREFLIGHT_VERSION
)
R7_PREFLIGHT_GATE_PATH = (
    R7_PREFLIGHT_ROOT / 'r7_local_contrastive_preflight_gate.json'
)
R7_POLICY_PATH = R7_PREFLIGHT_ROOT / 'r7_local_contrastive_policy.json'
R5_PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / R5_PILOT_VERSION
R5_RESULT_GATE_PATH = R5_PILOT_ROOT / 'pilot_result_gate.json'
R5_BEST_CHECKPOINT_PATH = R5_PILOT_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'
R7_R5_BASELINE_DIAGNOSTIC_PATH = (
    PILOT_DIAGNOSTIC_ROOT / 'r5_volume_bounded_baseline.json'
)


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH, R7_PREFLIGHT_GATE_PATH, R7_POLICY_PATH,
    R5_RESULT_GATE_PATH, R5_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
r7_preflight_gate = load_json(R7_PREFLIGHT_GATE_PATH)
r7_policy = load_json(R7_POLICY_PATH)
r5_result_gate = load_json(R5_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r7_component_preflight_passed': (
        r7_preflight_gate.get('preflight_passed') is True
    ),
    'r7_component_preflight_all_checks_true': (
        len(r7_preflight_gate.get('checks', {})) > 0
        and all(bool(value) for value in r7_preflight_gate['checks'].values())
    ),
    'r7_preflight_version_matches': (
        r7_preflight_gate.get('preflight_version') == R7_PREFLIGHT_VERSION
    ),
    'r7_preflight_revision_matches': (
        r7_preflight_gate.get('code_revision')
        == R7_PREFLIGHT_CODE_REVISION
    ),
    'r7_implementation_allowed': (
        r7_preflight_gate.get('r7_pilot_implementation_allowed') is True
    ),
    'r7_training_was_locked_before_this_block': (
        r7_preflight_gate.get('r7_pilot_training_authorized') is False
    ),
    'r7_architecture_parameter_count_unchanged': (
        int(r7_preflight_gate.get('architecture_parameters', -1))
        == EXPECTED_V22_PARAMETERS
        and int(r7_preflight_gate.get('new_trainable_parameters', -1)) == 0
    ),
    'r7_preflight_target_scope_is_fit_only': (
        r7_preflight_gate.get('target_arrays_read_splits') == ['fold0_fit']
        and r7_preflight_gate.get('fold0_heldout_targets_read') is False
        and r7_preflight_gate.get('locked_validation_targets_read') is False
        and r7_preflight_gate.get('test_targets_read') is False
    ),
    'r7_policy_path_matches_gate': (
        str(r7_preflight_gate.get('policy_path')) == str(R7_POLICY_PATH)
    ),
    'r7_policy_local_contrastive_contract_matches': (
        r7_policy.get('selected_component_objective', {}).get(
            'eligible_tiers'
        ) == sorted(R7_SMALL_COMPONENT_TIERS)
        and int(r7_policy.get('selected_component_objective', {}).get(
            'start_epoch', -1
        )) == R7_COMPONENT_LOSS_START_EPOCH
        and int(r7_policy.get('selected_component_objective', {}).get(
            'full_coefficient_epoch', -1
        )) == R7_COMPONENT_LOSS_FULL_EPOCH
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('maximum_coefficient', float('nan'))),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('inner_radius_mm', float('nan'))),
            R7_INNER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('outer_radius_mm', float('nan'))),
            R7_OUTER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('positive_weight', float('nan'))),
            R7_POSITIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_weight', float('nan'))),
            R7_HARD_NEGATIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('rank_weight', float('nan'))),
            R7_RANK_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_ratio', -1)) == R7_HARD_NEGATIVE_RATIO
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_minimum', -1)) == R7_HARD_NEGATIVE_MINIMUM
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('logit_margin', float('nan'))),
            R7_LOGIT_MARGIN, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get('experiment_seed', -1)) == PILOT_SEED
    ),
    'r7_policy_requires_volume_bounded_diagnostics': (
        r7_policy.get('proposal_diagnostics', {}).get(
            'raw_distance_metrics_are_not_success_gates'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_proposal_precision'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_predicted_to_target_volume_ratio'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_voxel_average_precision'
        ) is True
        and math.isclose(float(r7_policy.get(
            'proposal_diagnostics', {}
        ).get('volume_budget_multiplier', float('nan'))),
            R7_VOLUME_BUDGET_MULTIPLIER, rel_tol=0.0, abs_tol=1e-12)
    ),
    'r7_policy_preserves_whole_mask_loss': (
        math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('soft_dice_weight', float('nan'))),
            0.65, rel_tol=0.0, abs_tol=1e-12,
        )
        and math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('focal_bce_weight', float('nan'))),
            0.35, rel_tol=0.0, abs_tol=1e-12,
        )
    ),
    'r7_checkpoint_policy_is_decoupled': (
        r7_policy.get('checkpoint_policy', {}).get('save_checkpoint_on')
        == 'every strict numeric score improvement'
        and r7_policy.get('checkpoint_policy', {}).get('minimum_delta_use')
        == 'early-stopping patience reset only'
        and r7_policy.get('checkpoint_policy', {}).get(
            'minimum_delta_must_not_block_checkpoint_save'
        ) is True
    ),
    'r5_read_only_baseline_gate_matches': (
        r5_result_gate.get('pilot_version') == R5_PILOT_VERSION
        and r5_result_gate.get('code_revision') == R5_PILOT_CODE_REVISION
        and r5_result_gate.get('pilot_complete') is True
        and str(r5_result_gate.get('best_checkpoint_path'))
        == str(R5_BEST_CHECKPOINT_PATH)
        and r5_result_gate.get('best_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def r7_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


R7_INNER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_INNER_RADIUS_MM
)
R7_OUTER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_OUTER_RADIUS_MM
)


def r7_component_coefficient(epoch):
    epoch = int(epoch)
    if epoch < R7_COMPONENT_LOSS_START_EPOCH:
        return 0.0
    progress = (
        (epoch - R7_COMPONENT_LOSS_START_EPOCH + 1)
        / (R7_COMPONENT_LOSS_FULL_EPOCH - R7_COMPONENT_LOSS_START_EPOCH + 1)
    )
    return float(
        R7_COMPONENT_LOSS_MAX_COEFFICIENT * np.clip(progress, 0.0, 1.0)
    )


R7_COMPONENT_COEFFICIENT_SCHEDULE = {
    epoch: r7_component_coefficient(epoch)
    for epoch in range(1, MAX_EPOCHS + 1)
}
R7_EXPECTED_RAMP = {
    8: 0.02, 9: 0.04, 10: 0.06, 11: 0.08, 12: 0.10,
}
if not all(
    R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch] == 0.0
    for epoch in range(1, R7_COMPONENT_LOSS_START_EPOCH)
):
    raise RuntimeError('R7 local-contrastive coefficient is nonzero before epoch 8.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch], expected,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch, expected in R7_EXPECTED_RAMP.items()
):
    raise RuntimeError('R7 local-contrastive coefficient ramp changed.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch],
        R7_COMPONENT_LOSS_MAX_COEFFICIENT,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch in range(R7_COMPONENT_LOSS_FULL_EPOCH, MAX_EPOCHS + 1)
):
    raise RuntimeError('R7 local-contrastive coefficient does not remain capped.')


def r7_component_regions(exact_target, full_target, valid_domain):
    exact = np.asarray(exact_target, dtype=bool)
    full = np.asarray(full_target) > 0
    domain = np.asarray(valid_domain, dtype=bool)
    inner = ndimage.binary_dilation(exact, structure=R7_INNER_STRUCTURE)
    outer = ndimage.binary_dilation(exact, structure=R7_OUTER_STRUCTURE)
    negative = outer & ~inner & domain & ~full
    return inner, outer, negative


def pilot_extract_targets(
    row, local_origin, sample_type, atomic_component_id, component_tier
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary > 0
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(binary, dtype=np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if start_a != start_b or start_a != start_c:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_domain = F.max_pool3d(
        torch.from_numpy(domain_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0

    selected_sample = str(sample_type) == 'positive'
    selected_valid = (
        selected_sample and str(component_tier) in R7_SMALL_COMPONENT_TIERS
    )
    selected_component = np.zeros_like(component_support, dtype=bool)
    if selected_sample:
        labels, count = ndimage.label(
            component_support, structure=np.ones((3, 3, 3), dtype=bool)
        )
        expected_count = len(component_records.get(subject_id, []))
        if count != expected_count:
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {expected_count}.'
            )
        component_id = int(atomic_component_id)
        if component_id < 1 or component_id > count:
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is outside '
                f'the frozen 1..{count} range.'
            )
        selected_component = labels == component_id
        if not bool(selected_component.any()):
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is empty.'
            )

    selected_canvas, selected_start = symmetric_pad(
        selected_component.astype(np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if selected_start != start_a:
        raise RuntimeError(
            f'{subject_id}: selected-component padding start disagrees.'
        )
    selected_global = F.max_pool3d(
        torch.from_numpy(selected_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0
    selected_negative = np.zeros_like(selected_global, dtype=bool)
    if selected_valid:
        _, _, selected_negative = r7_component_regions(
            selected_global, global_target[0], global_domain
        )
        if not bool(selected_negative.any()):
            raise RuntimeError(
                f'{subject_id}: selected component has no lesion-free '
                '4-12 mm negative annulus.'
            )
    if selected_sample:
        frozen = [
            item for item in component_records[subject_id]
            if int(item['atomic_component_id']) == int(atomic_component_id)
        ]
        if len(frozen) != 1:
            raise RuntimeError(
                f'{subject_id}: selected component does not map uniquely to '
                'the frozen component audit.'
            )
        expected_global_voxels = int(frozen[0]['global_proposal_voxels'])
        if int(selected_global.sum()) != expected_global_voxels:
            raise RuntimeError(
                f'{subject_id}: selected component has '
                f'{int(selected_global.sum())} pooled voxels; expected '
                f'{expected_global_voxels}.'
            )
        if bool((selected_negative & selected_global).any()):
            raise RuntimeError(
                f'{subject_id}: negative annulus overlaps exact component.'
            )
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'selected_component_target': selected_global[None].astype(bool),
        'selected_component_negative_annulus': (
            selected_negative[None].astype(bool)
        ),
        'selected_component_valid': np.asarray(
            selected_valid, dtype=bool
        ),
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'selected_component_target': targets[
                'selected_component_target'
            ],
            'selected_component_negative_annulus': targets[
                'selected_component_negative_annulus'
            ],
            'selected_component_valid': targets[
                'selected_component_valid'
            ],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def r7_local_contrastive_loss(
    logits, exact_component_target, negative_annulus_target, component_valid,
):
    exact_target = exact_component_target.bool()
    negative_target = negative_annulus_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    positive_losses = []
    negative_losses = []
    rank_losses = []
    positive_logits = []
    hard_negative_logits = []
    selected_negative_counts = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]].float()
        negative = logits[batch_index][negative_target[batch_index]].float()
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError(
                'A valid r7 component must have positive voxels and a '
                'lesion-free negative annulus.'
            )
        hard_count = min(
            int(negative.numel()),
            max(
                R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * int(positive.numel()),
            ),
        )
        hard_negative = torch.topk(
            negative, k=hard_count, largest=True
        ).values
        positive_loss = F.softplus(-positive).mean()
        negative_loss = F.softplus(hard_negative).mean()
        rank_loss = F.softplus(
            R7_LOGIT_MARGIN - positive.mean() + hard_negative.mean()
        )
        contrastive_loss = (
            R7_POSITIVE_WEIGHT * positive_loss
            + R7_HARD_NEGATIVE_WEIGHT * negative_loss
            + R7_RANK_WEIGHT * rank_loss
        )
        losses.append(contrastive_loss)
        positive_losses.append(positive_loss)
        negative_losses.append(negative_loss)
        rank_losses.append(rank_loss)
        positive_logits.append(positive.mean())
        hard_negative_logits.append(hard_negative.mean())
        selected_negative_counts.append(hard_count)

    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'local_contrastive_loss': zero,
            'component_positive_loss': zero,
            'component_hard_negative_loss': zero,
            'component_rank_loss': zero,
            'mean_component_positive_logit': zero.detach(),
            'mean_component_hard_negative_logit': zero.detach(),
            'selected_hard_negative_voxels': zero.detach(),
            'selected_components_in_batch': zero.detach(),
        }
    return torch.stack(losses).mean(), {
        'local_contrastive_loss': torch.stack(losses).mean(),
        'component_positive_loss': torch.stack(positive_losses).mean(),
        'component_hard_negative_loss': torch.stack(negative_losses).mean(),
        'component_rank_loss': torch.stack(rank_losses).mean(),
        'mean_component_positive_logit': torch.stack(positive_logits).mean(),
        'mean_component_hard_negative_logit': torch.stack(
            hard_negative_logits
        ).mean(),
        'selected_hard_negative_voxels': torch.as_tensor(
            float(sum(selected_negative_counts)), device=logits.device
        ),
        'selected_components_in_batch': torch.as_tensor(
            float(len(losses)), device=logits.device
        ),
    }


def r7_local_contrastive_proposal_loss(
    logits, full_target, exact_component_target,
    negative_annulus_target, component_valid, epoch,
):
    whole_dice = pilot_soft_dice_loss(logits, full_target).mean()
    whole_bce = pilot_focal_bce_loss(logits, full_target).mean()
    whole_loss = 0.65 * whole_dice + 0.35 * whole_bce
    contrastive_loss, contrastive_parts = r7_local_contrastive_loss(
        logits, exact_component_target, negative_annulus_target,
        component_valid,
    )
    coefficient = r7_component_coefficient(epoch)
    proposal_loss = whole_loss + coefficient * contrastive_loss
    return proposal_loss, {
        'proposal_whole_mask_loss': whole_loss,
        'proposal_dice_loss': whole_dice,
        'proposal_focal_bce_loss': whole_bce,
        **contrastive_parts,
        'component_loss_coefficient': torch.as_tensor(
            coefficient, device=logits.device, dtype=torch.float32
        ),
    }


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(
    model, cpu_batch, device, epoch, batch_index,
    dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    if dropout_flags_override is not None:
        required_flags = {
            'force_atlas_dropout', 'force_vascular_dropout',
            'force_occlusion_dropout', 'force_clinical_dropout',
            'force_global_dropout', 'force_arterial_dropout',
        }
        if set(dropout_flags_override) != required_flags:
            raise RuntimeError(
                'Dropout override keys do not match the frozen branch set.'
            )
        flags = {
            name: bool(dropout_flags_override[name])
            for name in sorted(required_flags)
        }
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_loss, proposal_parts = r7_local_contrastive_proposal_loss(
        output['global_proposal_logits'], global_target,
        batch['selected_component_target'],
        batch['selected_component_negative_annulus'],
        batch['selected_component_valid'], epoch,
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        **proposal_parts,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'arterial_valid_nodes': arterial_valid.sum().detach().float(),
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'arterial_dropped': torch.as_tensor(float(flags['force_arterial_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_volume_bounded_prediction(probability, target_voxels):
    probability = np.asarray(probability, dtype=np.float32)
    budget = min(
        probability.size,
        max(1, int(math.ceil(
            R7_VOLUME_BUDGET_MULTIPLIER * int(target_voxels)
        ))),
    )
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape), budget


def pilot_histogram_average_precision(positive_histogram, negative_histogram):
    positive = np.asarray(positive_histogram, dtype=np.float64)[::-1]
    negative = np.asarray(negative_histogram, dtype=np.float64)[::-1]
    total_positive = float(positive.sum())
    if total_positive <= 0:
        return float('nan')
    cumulative_positive = np.cumsum(positive)
    cumulative_total = np.cumsum(positive + negative)
    precision = cumulative_positive / np.maximum(cumulative_total, 1.0)
    return float((precision * positive).sum() / total_positive)


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    voxel_precision = total_intersection / max(total_prediction, 1)
    predicted_to_target_volume_ratio = (
        total_prediction / max(total_target, 1)
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'voxel_precision': float(voxel_precision),
        'predicted_to_target_volume_ratio': float(
            predicted_to_target_volume_ratio
        ),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['volume_calibration_score'] = np.exp(-np.abs(
        np.log(np.maximum(
            proposal_threshold_df['predicted_to_target_volume_ratio'], 1e-6
        ))
    ))
    proposal_threshold_df['proposal_selection_score'] = (
        0.35 * proposal_threshold_df['positive_global_dice']
        + 0.15 * proposal_threshold_df['lesion_f1']
        + 0.20 * proposal_threshold_df['voxel_precision']
        + 0.15 * proposal_threshold_df['component_recall']
        + 0.15 * proposal_threshold_df['volume_calibration_score']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        [
            'proposal_selection_score', 'positive_global_dice',
            'voxel_precision', 'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    proposal_ap_positive_histogram = np.zeros(256, dtype=np.int64)
    proposal_ap_negative_histogram = np.zeros(256, dtype=np.int64)
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        proposal_intersection = int((proposal_prediction & target).sum())
        proposal_predicted_voxels = int(proposal_prediction.sum())
        proposal_distance = None
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        budgeted_proposal_prediction = None
        budgeted_proposal_distance = None
        proposal_volume_budget_voxels = 0
        if hard_eligible and target_voxels > 0:
            (
                budgeted_proposal_prediction,
                proposal_volume_budget_voxels,
            ) = pilot_volume_bounded_prediction(proposal, target_voxels)
            budgeted_proposal_distance = ndimage.distance_transform_edt(
                ~budgeted_proposal_prediction, sampling=case['spacing']
            )
            proposal_ap_positive_histogram += np.histogram(
                proposal[target], bins=256, range=(0.0, 1.0)
            )[0]
            proposal_ap_negative_histogram += np.histogram(
                proposal[~target], bins=256, range=(0.0, 1.0)
            )[0]
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
            'proposal_predicted_voxels': proposal_predicted_voxels,
            'proposal_intersection_voxels': proposal_intersection,
            'proposal_voxel_precision': (
                proposal_intersection / max(proposal_predicted_voxels, 1)
            ),
            'proposal_voxel_recall': (
                proposal_intersection / max(target_voxels, 1)
            ),
            'proposal_predicted_to_target_volume_ratio': (
                proposal_predicted_voxels / max(target_voxels, 1)
            ),
            'proposal_volume_budget_voxels': proposal_volume_budget_voxels,
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            budgeted_proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    budgeted_proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                budgeted_proposal_max_iou = float(
                    budgeted_proposal_iou[component_id][
                        'maximum_component_iou'
                    ]
                )
                proposal_distance_mm = float('nan')
                if proposal_distance is not None:
                    proposal_distance_mm = float(
                        proposal_distance[component_slice][component_local].min(
                            initial=np.inf
                        )
                    )
                budgeted_proposal_distance_mm = float(
                    budgeted_proposal_distance[component_slice][
                        component_local
                    ].min(initial=np.inf)
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                budgeted_proposal_covered = (
                    budgeted_proposal_max_iou >= 0.05
                )
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += int(budgeted_proposal_covered)
                tier_counts[tier][3] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'proposal_minimum_distance_mm': proposal_distance_mm,
                    'budgeted_proposal_maximum_component_iou': (
                        budgeted_proposal_max_iou
                    ),
                    'budgeted_proposal_minimum_distance_mm': (
                        budgeted_proposal_distance_mm
                    ),
                    'proposal_within_2mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 2.0
                    ),
                    'proposal_within_4mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 4.0
                    ),
                    'proposal_within_8mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_within_2mm': bool(
                        budgeted_proposal_distance_mm <= 2.0
                    ),
                    'budgeted_proposal_within_4mm': bool(
                        budgeted_proposal_distance_mm <= 4.0
                    ),
                    'budgeted_proposal_within_8mm': bool(
                        budgeted_proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_covered': bool(
                        budgeted_proposal_covered
                    ),
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, budgeted_proposal_tp, total = tier_counts[tier]
        tier_components = [
            item for item in component_rows
            if item['component_tier'] == tier
        ]
        finite_distances = [
            float(item['proposal_minimum_distance_mm'])
            for item in tier_components
            if np.isfinite(item['proposal_minimum_distance_mm'])
        ]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
            'budgeted_proposal_component_recall': (
                budgeted_proposal_tp / max(total, 1)
            ),
            'proposal_within_2mm_fraction': float(np.mean([
                item['proposal_within_2mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_4mm_fraction': float(np.mean([
                item['proposal_within_4mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_8mm_fraction': float(np.mean([
                item['proposal_within_8mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_2mm_fraction': float(np.mean([
                item['budgeted_proposal_within_2mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_4mm_fraction': float(np.mean([
                item['budgeted_proposal_within_4mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_8mm_fraction': float(np.mean([
                item['budgeted_proposal_within_8mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'median_finite_proposal_distance_mm': float(
                np.median(finite_distances)
            ) if finite_distances else float('nan'),
            'mean_proposal_maximum_probability': float(np.mean([
                item['proposal_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'mean_final_maximum_probability': float(np.mean([
                item['final_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'representation_failure_fraction': float(np.mean([
                item['representation_failure_below_0p10']
                for item in tier_components
            ])) if tier_components else 0.0,
        })
    small_component_rows = [
        item for item in component_rows
        if item['component_tier'] in set(COMPONENT_TIERS[:4])
    ]
    proposal_voxel_average_precision = pilot_histogram_average_precision(
        proposal_ap_positive_histogram, proposal_ap_negative_histogram
    )
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'proposal_voxel_precision': float(
            proposal_selected['voxel_precision']
        ),
        'proposal_predicted_to_target_volume_ratio': float(
            proposal_selected['predicted_to_target_volume_ratio']
        ),
        'proposal_volume_calibration_score': float(
            proposal_selected['volume_calibration_score']
        ),
        'proposal_voxel_average_precision_256bin': float(
            proposal_voxel_average_precision
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'small_components_audited': len(small_component_rows),
        'small_component_proposal_within_2mm_fraction': float(np.mean([
            item['proposal_within_2mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_4mm_fraction': float(np.mean([
            item['proposal_within_4mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_8mm_fraction': float(np.mean([
            item['proposal_within_8mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_budgeted_proposal_within_2mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_2mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_4mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_4mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_8mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_8mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered'] for item in component_rows
        ])) if component_rows else 0.0,
        'small_budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_representation_failure_fraction': float(np.mean([
            item['representation_failure_below_0p10']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    required_finite_summary_metrics = [
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'proposal_global_dice', 'proposal_component_recall',
        'proposal_voxel_precision',
        'proposal_predicted_to_target_volume_ratio',
        'proposal_voxel_average_precision_256bin',
        'budgeted_proposal_component_recall',
        'small_budgeted_proposal_component_recall',
        'small_component_budgeted_proposal_within_2mm_fraction',
        'small_component_budgeted_proposal_within_4mm_fraction',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'small_component_representation_failure_fraction',
        'pilot_checkpoint_score',
    ]
    nonfinite_summary_metrics = [
        name for name in required_finite_summary_metrics
        if not np.isfinite(float(summary[name]))
    ]
    if nonfinite_summary_metrics:
        raise RuntimeError(
            'Validation produced non-finite required metrics: '
            + ', '.join(nonfinite_summary_metrics)
        )
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
            and completed.get('best_checkpoint_sha256')
            == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R7 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_index = None
    smoke_subject_id = None
    for candidate_index, entry in enumerate(dataset.schedule):
        if entry['sample_type'] != 'positive':
            continue
        candidate_subject = str(entry['subject_id'])
        if not as_bool(
            fit_row_by_subject[candidate_subject]['trial8_atlas_enabled']
        ):
            continue
        candidate_sample = dataset[candidate_index]
        candidate_checks = {
            'atlas_enabled': bool(np.asarray(
                candidate_sample['atlas_enabled']
            ).reshape(-1)[0]),
            'territory_nodes_valid': bool(np.asarray(
                candidate_sample['territory_valid']
            ).astype(bool).any()),
            'vessel_nodes_valid': bool(np.asarray(
                candidate_sample['vessel_valid']
            ).astype(bool).any()),
            'selected_component_valid': bool(np.asarray(
                candidate_sample['selected_component_valid']
            ).reshape(-1)[0]),
        }
        if all(candidate_checks.values()):
            smoke_index = int(candidate_index)
            smoke_subject_id = candidate_subject
            break
    if smoke_index is None:
        raise RuntimeError(
            'No positive fold-0-fit sample has simultaneous atlas, territory, '
            'vessel, and selected-component support for the all-branch smoke '
            'test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    smoke_dropout_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device,
            epoch=R7_COMPONENT_LOSS_FULL_EPOCH, batch_index=0,
            dropout_flags_override=smoke_dropout_flags,
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    r7_smoke_checks = {
        'positive_sample_selected': bool(
            smoke_batch['selected_component_valid'].bool().all().item()
        ),
        'atlas_enabled_for_smoke': bool(
            smoke_batch['atlas_enabled'].bool().all().item()
        ),
        'territory_support_for_smoke': bool(
            smoke_batch['territory_valid'].bool().any().item()
        ),
        'vessel_support_for_smoke': bool(
            smoke_batch['vessel_valid'].bool().any().item()
        ),
        'projected_arterial_nodes_valid_for_smoke': float(
            smoke_parts['arterial_valid_nodes'].detach().float().cpu()
        ) > 0.0,
        'arterial_loss_finite_for_smoke': bool(torch.isfinite(
            smoke_parts['arterial_loss']
        ).item()),
        'all_branch_dropouts_disabled_for_smoke': not any(
            smoke_dropout_flags.values()
        ),
        'exact_component_target_nonempty': bool(
            smoke_batch['selected_component_target'].bool().any().item()
        ),
        'negative_annulus_nonempty': bool(
            smoke_batch['selected_component_negative_annulus']
            .bool().any().item()
        ),
        'negative_annulus_disjoint_from_exact_component': not bool(torch.any(
            smoke_batch['selected_component_target'].bool()
            & smoke_batch['selected_component_negative_annulus'].bool()
        ).item()),
        'local_contrastive_loss_positive': float(
            smoke_parts['local_contrastive_loss']
            .detach().float().cpu()
        ) > 0.0,
        'smoke_uses_maximum_contrastive_coefficient': math.isclose(
            float(smoke_parts['component_loss_coefficient']
                  .detach().float().cpu()),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT,
            rel_tol=0.0, abs_tol=1e-7,
        ),
        'whole_mask_proposal_loss_finite': bool(torch.isfinite(
            smoke_parts['proposal_whole_mask_loss']
        ).item()),
    }
    if not all(r7_smoke_checks.values()):
        failed = [
            name for name, passed in r7_smoke_checks.items() if not passed
        ]
        raise RuntimeError(
            'R7 local-contrastive smoke checks failed: ' + ', '.join(failed)
        )
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    if R7_R5_BASELINE_DIAGNOSTIC_PATH.exists():
        r5_baseline_diagnostic = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
        if (
            r5_baseline_diagnostic.get('r5_checkpoint_sha256')
            != r5_checkpoint_sha256
            or r5_baseline_diagnostic.get('metric_contract_revision')
            != PILOT_CODE_REVISION
        ):
            raise RuntimeError(
                'Existing r5 volume-bounded baseline is incompatible with '
                'the current r7 contract.'
            )
        r5_recomputed_summary = dict(
            r5_baseline_diagnostic['validation_summary']
        )
    else:
        rng_state_before_baseline = pilot_capture_rng_state()
        r5_checkpoint = torch.load(
            R5_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
        )
        r5_checkpoint_checks = {
            'pilot_version': (
                r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
            ),
            'code_revision': (
                r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
            ),
            'parameter_count': int(
                r5_checkpoint.get('parameter_count', -1)
            ) == EXPECTED_V22_PARAMETERS,
        }
        if not all(r5_checkpoint_checks.values()):
            failed = [
                name for name, passed in r5_checkpoint_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Unsafe r5 read-only baseline checkpoint: '
                + ', '.join(failed)
            )
        r5_baseline_model = VascuFateNetV22(pilot_schema).to(device).eval()
        r5_baseline_model.load_state_dict(r5_checkpoint['ema_model_state'])
        for parameter in r5_baseline_model.parameters():
            parameter.requires_grad_(False)
        print(
            '\nRecomputing the completed r5 best checkpoint with the r7 '
            'precision- and volume-bounded diagnostic contract.'
        )
        r5_recomputed_summary, r5_recomputed_tiers = pilot_validate(
            r5_baseline_model, epoch=0, device=device
        )
        r5_baseline_diagnostic = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'metric_contract_revision': PILOT_CODE_REVISION,
            'r5_pilot_version': R5_PILOT_VERSION,
            'r5_code_revision': R5_PILOT_CODE_REVISION,
            'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'validation_summary': r5_recomputed_summary,
            'component_tier_metrics': r5_recomputed_tiers,
            'r5_checkpoint_loaded_into_r7_model': False,
            'optimizer_steps': 0,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        }
        atomic_write_json(
            r5_baseline_diagnostic, R7_R5_BASELINE_DIAGNOSTIC_PATH
        )
        del r5_baseline_model, r5_checkpoint
        gc.collect()
        torch.cuda.empty_cache()
        pilot_restore_rng_state(rng_state_before_baseline)

    
    
    
    
    del model, ema_model
    gc.collect()
    torch.cuda.empty_cache()
    set_determinism(PILOT_SEED)
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed during lineage reset.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    lineage_smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True
    )
    lineage_smoke_batch = next(iter(lineage_smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        lineage_smoke_loss, lineage_smoke_parts = (
            pilot_compute_training_loss(
                model, lineage_smoke_batch, device, epoch=1, batch_index=0
            )
        )
    if not bool(torch.isfinite(lineage_smoke_loss).item()):
        raise RuntimeError('R7 controlled-lineage smoke loss is non-finite.')
    if float(
        lineage_smoke_parts['component_loss_coefficient']
        .detach().float().cpu()
    ) != 0.0:
        raise RuntimeError(
            'R7 controlled-lineage epoch-1 coefficient is not exactly zero.'
        )
    lineage_smoke_loss.backward()
    lineage_gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            parameter.grad
            for name, parameter in model.named_parameters()
            if name.startswith(prefix) and parameter.grad is not None
        ]
        lineage_gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(gradient).all().item())
            for gradient in gradients
        ) and any(
            float(gradient.detach().float().norm().cpu()) > 0.0
            for gradient in gradients
        )
    model.zero_grad(set_to_none=True)
    if not all(lineage_gradient_checks.values()):
        failed = [
            name for name, passed in lineage_gradient_checks.items()
            if not passed
        ]
        raise RuntimeError(
            'R7 controlled-lineage gradient smoke failed: '
            + ', '.join(failed)
        )

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r7_last_checkpoint_only',
        'strict_best_checkpoint_policy': (
            'save every strict numeric pilot-score improvement'
        ),
        'minimum_delta_policy': (
            'used only to reset early-stopping patience'
        ),
        'r7_preflight_gate_sha256': file_sha256(R7_PREFLIGHT_GATE_PATH),
        'r7_proposal_loss_policy_sha256': file_sha256(R7_POLICY_PATH),
        'component_loss_start_epoch': R7_COMPONENT_LOSS_START_EPOCH,
        'component_loss_full_epoch': R7_COMPONENT_LOSS_FULL_EPOCH,
        'component_loss_maximum_coefficient': (
            R7_COMPONENT_LOSS_MAX_COEFFICIENT
        ),
        'component_positive_weight': R7_POSITIVE_WEIGHT,
        'component_hard_negative_weight': R7_HARD_NEGATIVE_WEIGHT,
        'component_rank_weight': R7_RANK_WEIGHT,
        'component_inner_radius_mm': R7_INNER_RADIUS_MM,
        'component_outer_radius_mm': R7_OUTER_RADIUS_MM,
        'component_logit_margin': R7_LOGIT_MARGIN,
        'component_eligible_tiers': sorted(R7_SMALL_COMPONENT_TIERS),
        'component_coefficient_schedule': {
            str(epoch): coefficient
            for epoch, coefficient in R7_COMPONENT_COEFFICIENT_SCHEDULE.items()
        },
        'proposal_volume_budget_multiplier': R7_VOLUME_BUDGET_MULTIPLIER,
        'proposal_volume_budget_policy': (
            'diagnostic only: retain the top 3x ground-truth target voxels; '
            'never used as an inference rule or checkpoint-selection input'
        ),
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'controlled_lineage_matches_r5_seed': True,
        'controlled_lineage_epoch1_component_coefficient': 0.0,
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'controlled_lineage_gradient_checks': lineage_gradient_checks,
        'controlled_lineage_smoke_loss': float(
            lineage_smoke_loss.detach().cpu()
        ),
        'r7_selected_component_smoke_checks': r7_smoke_checks,
        'r7_component_coefficient_schedule_verified': True,
        'r7_smoke_subject_id': smoke_subject_id,
        'r7_smoke_schedule_index': smoke_index,
        'r7_smoke_dropout_flags': smoke_dropout_flags,
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R7 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R7 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r7 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Controlled r5-seed lineage:          verified')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('Local contrastive rescue:              small tiers, epochs 8-40')
    print('Maximum contrastive coefficient:       0.10')
    print('Proposal anti-shortcut diagnostics:    oracle-volume-bounded (audit only)')
    print('R5 recomputed mean/global Dice:        '
          f'{r5_recomputed_summary["positive_mean_dice"]:.4f} / '
          f'{r5_recomputed_summary["positive_global_dice"]:.4f}')
    print('R5 oracle-budgeted small within 8 mm:  '
          f'{r5_recomputed_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}')
    print('Best-checkpoint minimum delta:        disabled (strict numeric best)')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        strict_best_improved = current_score > best_score
        patience_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if strict_best_improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
        if patience_improved:
            patience_reference_score = current_score
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        epoch_tier_metrics = {
            str(row['component_tier']): row for row in tier_rows
        }
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'proposal_voxel_precision': summary['proposal_voxel_precision'],
            'proposal_predicted_to_target_volume_ratio': summary[
                'proposal_predicted_to_target_volume_ratio'
            ],
            'proposal_voxel_average_precision_256bin': summary[
                'proposal_voxel_average_precision_256bin'
            ],
            'budgeted_proposal_component_recall': summary[
                'budgeted_proposal_component_recall'
            ],
            'small_component_proposal_within_2mm_fraction': summary[
                'small_component_proposal_within_2mm_fraction'
            ],
            'small_component_proposal_within_4mm_fraction': summary[
                'small_component_proposal_within_4mm_fraction'
            ],
            'small_component_proposal_within_8mm_fraction': summary[
                'small_component_proposal_within_8mm_fraction'
            ],
            'small_component_budgeted_proposal_within_8mm_fraction': summary[
                'small_component_budgeted_proposal_within_8mm_fraction'
            ],
            'small_component_representation_failure_fraction': summary[
                'small_component_representation_failure_fraction'
            ],
            **{
                f'{tier_name}_{metric_name}': float(
                    epoch_tier_metrics.get(tier_name, {}).get(
                        metric_name, float('nan')
                    )
                )
                for tier_name in COMPONENT_TIERS
                for metric_name in (
                    'final_component_recall',
                    'proposal_component_recall',
                    'budgeted_proposal_component_recall',
                    'proposal_within_8mm_fraction',
                    'budgeted_proposal_within_8mm_fraction',
                    'representation_failure_fraction',
                )
            },
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'strict_best_improved': strict_best_improved,
            'patience_improved_by_minimum_delta': patience_improved,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if strict_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'contrastive={running["local_contrastive_loss"]/max(processed,1):.4f}, '
            f'coef={running["component_loss_coefficient"]/max(processed,1):.3f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        print(
            '  R7 raw proposal geometry: small within 2/4/8 mm='
            f'{summary["small_component_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_8mm_fraction"]:.4f}, '
            'small representation failure='
            f'{summary["small_component_representation_failure_fraction"]:.4f}'
        )
        print(
            '  Oracle-volume-bounded diagnostic: small within 2/4/8 mm='
            f'{summary["small_component_budgeted_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}, '
            f'component recall={summary["budgeted_proposal_component_recall"]:.4f}, '
            f'precision/AP={summary["proposal_voxel_precision"]:.4f}/'
            f'{summary["proposal_voxel_average_precision_256bin"]:.4f}, '
            f'volume ratio={summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_baseline = r7_policy['r5_baseline']
    success_policy = r7_policy['r7_directional_success_gates']
    best_tier_by_name = {
        str(row['component_tier']): row for row in best_tier_rows
    }
    best_large_proposal_recall = float(
        best_tier_by_name.get('large', {}).get(
            'proposal_component_recall', 0.0
        )
    )
    mean_dice_improvement = (
        float(best_summary['positive_mean_dice'])
        - float(r5_recomputed_summary['positive_mean_dice'])
    )
    budgeted_small_within_8mm_improvement = (
        float(best_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
        - float(r5_recomputed_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
    )
    small_representation_failure_reduction = (
        float(r5_recomputed_summary[
            'small_component_representation_failure_fraction'
        ])
        - float(best_summary[
            'small_component_representation_failure_fraction'
        ])
    )
    global_dice_regression = (
        float(r5_recomputed_summary['positive_global_dice'])
        - float(best_summary['positive_global_dice'])
    )
    proposal_dice_regression = (
        float(r5_recomputed_summary['proposal_global_dice'])
        - float(best_summary['proposal_global_dice'])
    )
    r7_success_checks = {
        'mean_dice_does_not_materially_regress': (
            -mean_dice_improvement
            <= float(success_policy[
                'mean_dice_maximum_allowed_regression'
            ])
        ),
        'mean_dice_reaches_target_improvement': (
            mean_dice_improvement
            >= float(success_policy[
                'mean_dice_target_absolute_improvement'
            ])
        ),
        'volume_bounded_small_within_8mm_improves': (
            budgeted_small_within_8mm_improvement
            >= float(success_policy[
                'volume_bounded_small_within_8mm_absolute_improvement_at_least'
            ])
        ),
        'small_representation_failure_reduced_by_required_margin': (
            small_representation_failure_reduction
            >= float(success_policy[
                'small_representation_failure_absolute_reduction_at_least'
            ])
        ),
        'proposal_component_recall_meets_minimum': (
            float(best_summary['proposal_component_recall'])
            >= float(success_policy['proposal_component_recall_minimum'])
        ),
        'global_dice_regression_within_limit': (
            global_dice_regression
            <= float(success_policy[
                'global_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_dice_regression_within_limit': (
            proposal_dice_regression
            <= float(success_policy[
                'proposal_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_volume_ratio_is_bounded': (
            float(best_summary[
                'proposal_predicted_to_target_volume_ratio'
            ])
            <= float(success_policy[
                'proposal_predicted_to_target_volume_ratio_maximum'
            ])
        ),
        'empty_false_positive_volume_within_limit': (
            float(best_summary['empty_false_positive_volume_ml'])
            <= float(success_policy[
                'mean_false_positive_volume_ml_maximum'
            ])
        ),
        'large_component_proposal_recall_meets_minimum': (
            best_large_proposal_recall
            >= float(success_policy[
                'large_component_proposal_recall_minimum'
            ])
        ),
    }
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'r5_baseline_evidence': r5_baseline,
        'r5_recomputed_under_r7_metric_contract': r5_recomputed_summary,
        'r5_read_only_baseline_diagnostic_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r7_directional_success_policy': success_policy,
        'r7_directional_success_checks': r7_success_checks,
        'r7_directional_success_passed': all(r7_success_checks.values()),
        'r7_directional_effects': {
            'mean_dice_absolute_change': mean_dice_improvement,
            'volume_bounded_small_within_8mm_absolute_change': (
                budgeted_small_within_8mm_improvement
            ),
            'small_representation_failure_absolute_reduction': (
                small_representation_failure_reduction
            ),
            'global_dice_regression': global_dice_regression,
            'proposal_dice_regression': proposal_dice_regression,
            'proposal_predicted_to_target_volume_ratio': float(
                best_summary['proposal_predicted_to_target_volume_ratio']
            ),
            'proposal_voxel_average_precision_256bin': float(
                best_summary['proposal_voxel_average_precision_256bin']
            ),
            'large_component_proposal_recall': (
                best_large_proposal_recall
            ),
        },
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(
        'Raw small proposal within 8 mm:   '
        f'{best_summary["small_component_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Oracle-budgeted small within 8 mm: '
        f'{best_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Proposal precision / AP:          '
        f'{best_summary["proposal_voxel_precision"]:.4f} / '
        f'{best_summary["proposal_voxel_average_precision_256bin"]:.4f}'
    )
    print(
        'Proposal predicted/target ratio:  '
        f'{best_summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
    )
    print(
        'Mean Dice change vs recomputed r5:'
        f' {mean_dice_improvement:+.4f}'
    )
    print(
        'Budgeted within-8-mm change vs r5:'
        f' {budgeted_small_within_8mm_improvement:+.4f}'
    )
    print(
        'Small representation failure:     '
        f'{best_summary["small_component_representation_failure_fraction"]:.4f}'
    )
    print(
        'R7 directional success gates:     '
        f'{all(r7_success_checks.values())}'
    )
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


run_v22_fold0_pilot()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = (
    'r7_delayed_local_contrastive_volume_bounded_r1'
)
PILOT_VERSION = 'v2_2_fold0_local_contrastive_proposal_r7'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

R7_PREFLIGHT_VERSION = 'v1_delayed_small_component_local_contrastive'
R7_PREFLIGHT_CODE_REVISION = (
    'r1_precision_bounded_diffuse_shortcut_guard'
)
R7_COMPONENT_LOSS_MAX_COEFFICIENT = 0.10
R7_COMPONENT_LOSS_START_EPOCH = 8
R7_COMPONENT_LOSS_FULL_EPOCH = 12
R7_POSITIVE_WEIGHT = 0.40
R7_HARD_NEGATIVE_WEIGHT = 0.40
R7_RANK_WEIGHT = 0.20
R7_INNER_RADIUS_MM = 4.0
R7_OUTER_RADIUS_MM = 12.0
R7_HARD_NEGATIVE_RATIO = 4
R7_HARD_NEGATIVE_MINIMUM = 16
R7_LOGIT_MARGIN = 1.0
R7_VOLUME_BUDGET_MULTIPLIER = 3.0
R7_SMALL_COMPONENT_TIERS = set(COMPONENT_TIERS[:4])
R5_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
R5_PILOT_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
R7_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip((0.8, 0.8, 2.0), GLOBAL_DOWNSAMPLE)
)

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'
R7_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r7_local_contrastive_component_proposal' / R7_PREFLIGHT_VERSION
)
R7_PREFLIGHT_GATE_PATH = (
    R7_PREFLIGHT_ROOT / 'r7_local_contrastive_preflight_gate.json'
)
R7_POLICY_PATH = R7_PREFLIGHT_ROOT / 'r7_local_contrastive_policy.json'
R5_PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / R5_PILOT_VERSION
R5_RESULT_GATE_PATH = R5_PILOT_ROOT / 'pilot_result_gate.json'
R5_BEST_CHECKPOINT_PATH = R5_PILOT_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'
R7_R5_BASELINE_DIAGNOSTIC_PATH = (
    PILOT_DIAGNOSTIC_ROOT / 'r5_volume_bounded_baseline.json'
)
R7_FAILURE_DIAGNOSTIC_VERSION = 'v1_r5_r7_failure_mechanism_audit'
R7_FAILURE_DIAGNOSTIC_CODE_REVISION = (
    'r1_exposure_resolution_margin_volume_branch_checkpoint'
)
R7_FAILURE_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'model_diagnostics'
    / MODEL_VERSION / R7_FAILURE_DIAGNOSTIC_VERSION
)
R7_FAILURE_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
R7_FAILURE_GATE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_diagnostic_gate.json'
)
R7_FAILURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_summary.json'
)



PILOT_TARGET_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'fold0_target_access_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_schedule_audit.csv'
)
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH, R7_PREFLIGHT_GATE_PATH, R7_POLICY_PATH,
    R5_RESULT_GATE_PATH, R5_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
r7_preflight_gate = load_json(R7_PREFLIGHT_GATE_PATH)
r7_policy = load_json(R7_POLICY_PATH)
r5_result_gate = load_json(R5_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r7_component_preflight_passed': (
        r7_preflight_gate.get('preflight_passed') is True
    ),
    'r7_component_preflight_all_checks_true': (
        len(r7_preflight_gate.get('checks', {})) > 0
        and all(bool(value) for value in r7_preflight_gate['checks'].values())
    ),
    'r7_preflight_version_matches': (
        r7_preflight_gate.get('preflight_version') == R7_PREFLIGHT_VERSION
    ),
    'r7_preflight_revision_matches': (
        r7_preflight_gate.get('code_revision')
        == R7_PREFLIGHT_CODE_REVISION
    ),
    'r7_implementation_allowed': (
        r7_preflight_gate.get('r7_pilot_implementation_allowed') is True
    ),
    'r7_training_was_locked_before_this_block': (
        r7_preflight_gate.get('r7_pilot_training_authorized') is False
    ),
    'r7_architecture_parameter_count_unchanged': (
        int(r7_preflight_gate.get('architecture_parameters', -1))
        == EXPECTED_V22_PARAMETERS
        and int(r7_preflight_gate.get('new_trainable_parameters', -1)) == 0
    ),
    'r7_preflight_target_scope_is_fit_only': (
        r7_preflight_gate.get('target_arrays_read_splits') == ['fold0_fit']
        and r7_preflight_gate.get('fold0_heldout_targets_read') is False
        and r7_preflight_gate.get('locked_validation_targets_read') is False
        and r7_preflight_gate.get('test_targets_read') is False
    ),
    'r7_policy_path_matches_gate': (
        str(r7_preflight_gate.get('policy_path')) == str(R7_POLICY_PATH)
    ),
    'r7_policy_local_contrastive_contract_matches': (
        r7_policy.get('selected_component_objective', {}).get(
            'eligible_tiers'
        ) == sorted(R7_SMALL_COMPONENT_TIERS)
        and int(r7_policy.get('selected_component_objective', {}).get(
            'start_epoch', -1
        )) == R7_COMPONENT_LOSS_START_EPOCH
        and int(r7_policy.get('selected_component_objective', {}).get(
            'full_coefficient_epoch', -1
        )) == R7_COMPONENT_LOSS_FULL_EPOCH
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('maximum_coefficient', float('nan'))),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('inner_radius_mm', float('nan'))),
            R7_INNER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('outer_radius_mm', float('nan'))),
            R7_OUTER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('positive_weight', float('nan'))),
            R7_POSITIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_weight', float('nan'))),
            R7_HARD_NEGATIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('rank_weight', float('nan'))),
            R7_RANK_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_ratio', -1)) == R7_HARD_NEGATIVE_RATIO
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_minimum', -1)) == R7_HARD_NEGATIVE_MINIMUM
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('logit_margin', float('nan'))),
            R7_LOGIT_MARGIN, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get('experiment_seed', -1)) == PILOT_SEED
    ),
    'r7_policy_requires_volume_bounded_diagnostics': (
        r7_policy.get('proposal_diagnostics', {}).get(
            'raw_distance_metrics_are_not_success_gates'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_proposal_precision'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_predicted_to_target_volume_ratio'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_voxel_average_precision'
        ) is True
        and math.isclose(float(r7_policy.get(
            'proposal_diagnostics', {}
        ).get('volume_budget_multiplier', float('nan'))),
            R7_VOLUME_BUDGET_MULTIPLIER, rel_tol=0.0, abs_tol=1e-12)
    ),
    'r7_policy_preserves_whole_mask_loss': (
        math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('soft_dice_weight', float('nan'))),
            0.65, rel_tol=0.0, abs_tol=1e-12,
        )
        and math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('focal_bce_weight', float('nan'))),
            0.35, rel_tol=0.0, abs_tol=1e-12,
        )
    ),
    'r7_checkpoint_policy_is_decoupled': (
        r7_policy.get('checkpoint_policy', {}).get('save_checkpoint_on')
        == 'every strict numeric score improvement'
        and r7_policy.get('checkpoint_policy', {}).get('minimum_delta_use')
        == 'early-stopping patience reset only'
        and r7_policy.get('checkpoint_policy', {}).get(
            'minimum_delta_must_not_block_checkpoint_save'
        ) is True
    ),
    'r5_read_only_baseline_gate_matches': (
        r5_result_gate.get('pilot_version') == R5_PILOT_VERSION
        and r5_result_gate.get('code_revision') == R5_PILOT_CODE_REVISION
        and r5_result_gate.get('pilot_complete') is True
        and str(r5_result_gate.get('best_checkpoint_path'))
        == str(R5_BEST_CHECKPOINT_PATH)
        and r5_result_gate.get('best_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def r7_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


R7_INNER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_INNER_RADIUS_MM
)
R7_OUTER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_OUTER_RADIUS_MM
)


def r7_component_coefficient(epoch):
    epoch = int(epoch)
    if epoch < R7_COMPONENT_LOSS_START_EPOCH:
        return 0.0
    progress = (
        (epoch - R7_COMPONENT_LOSS_START_EPOCH + 1)
        / (R7_COMPONENT_LOSS_FULL_EPOCH - R7_COMPONENT_LOSS_START_EPOCH + 1)
    )
    return float(
        R7_COMPONENT_LOSS_MAX_COEFFICIENT * np.clip(progress, 0.0, 1.0)
    )


R7_COMPONENT_COEFFICIENT_SCHEDULE = {
    epoch: r7_component_coefficient(epoch)
    for epoch in range(1, MAX_EPOCHS + 1)
}
R7_EXPECTED_RAMP = {
    8: 0.02, 9: 0.04, 10: 0.06, 11: 0.08, 12: 0.10,
}
if not all(
    R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch] == 0.0
    for epoch in range(1, R7_COMPONENT_LOSS_START_EPOCH)
):
    raise RuntimeError('R7 local-contrastive coefficient is nonzero before epoch 8.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch], expected,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch, expected in R7_EXPECTED_RAMP.items()
):
    raise RuntimeError('R7 local-contrastive coefficient ramp changed.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch],
        R7_COMPONENT_LOSS_MAX_COEFFICIENT,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch in range(R7_COMPONENT_LOSS_FULL_EPOCH, MAX_EPOCHS + 1)
):
    raise RuntimeError('R7 local-contrastive coefficient does not remain capped.')


def r7_component_regions(exact_target, full_target, valid_domain):
    exact = np.asarray(exact_target, dtype=bool)
    full = np.asarray(full_target) > 0
    domain = np.asarray(valid_domain, dtype=bool)
    inner = ndimage.binary_dilation(exact, structure=R7_INNER_STRUCTURE)
    outer = ndimage.binary_dilation(exact, structure=R7_OUTER_STRUCTURE)
    negative = outer & ~inner & domain & ~full
    return inner, outer, negative


def pilot_extract_targets(
    row, local_origin, sample_type, atomic_component_id, component_tier
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary > 0
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(binary, dtype=np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if start_a != start_b or start_a != start_c:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_domain = F.max_pool3d(
        torch.from_numpy(domain_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0

    selected_sample = str(sample_type) == 'positive'
    selected_valid = (
        selected_sample and str(component_tier) in R7_SMALL_COMPONENT_TIERS
    )
    selected_component = np.zeros_like(component_support, dtype=bool)
    if selected_sample:
        labels, count = ndimage.label(
            component_support, structure=np.ones((3, 3, 3), dtype=bool)
        )
        expected_count = len(component_records.get(subject_id, []))
        if count != expected_count:
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {expected_count}.'
            )
        component_id = int(atomic_component_id)
        if component_id < 1 or component_id > count:
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is outside '
                f'the frozen 1..{count} range.'
            )
        selected_component = labels == component_id
        if not bool(selected_component.any()):
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is empty.'
            )

    selected_canvas, selected_start = symmetric_pad(
        selected_component.astype(np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if selected_start != start_a:
        raise RuntimeError(
            f'{subject_id}: selected-component padding start disagrees.'
        )
    selected_global = F.max_pool3d(
        torch.from_numpy(selected_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0
    selected_negative = np.zeros_like(selected_global, dtype=bool)
    if selected_valid:
        _, _, selected_negative = r7_component_regions(
            selected_global, global_target[0], global_domain
        )
        if not bool(selected_negative.any()):
            raise RuntimeError(
                f'{subject_id}: selected component has no lesion-free '
                '4-12 mm negative annulus.'
            )
    if selected_sample:
        frozen = [
            item for item in component_records[subject_id]
            if int(item['atomic_component_id']) == int(atomic_component_id)
        ]
        if len(frozen) != 1:
            raise RuntimeError(
                f'{subject_id}: selected component does not map uniquely to '
                'the frozen component audit.'
            )
        expected_global_voxels = int(frozen[0]['global_proposal_voxels'])
        if int(selected_global.sum()) != expected_global_voxels:
            raise RuntimeError(
                f'{subject_id}: selected component has '
                f'{int(selected_global.sum())} pooled voxels; expected '
                f'{expected_global_voxels}.'
            )
        if bool((selected_negative & selected_global).any()):
            raise RuntimeError(
                f'{subject_id}: negative annulus overlaps exact component.'
            )
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'selected_component_target': selected_global[None].astype(bool),
        'selected_component_negative_annulus': (
            selected_negative[None].astype(bool)
        ),
        'selected_component_valid': np.asarray(
            selected_valid, dtype=bool
        ),
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'selected_component_target': targets[
                'selected_component_target'
            ],
            'selected_component_negative_annulus': targets[
                'selected_component_negative_annulus'
            ],
            'selected_component_valid': targets[
                'selected_component_valid'
            ],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def r7_local_contrastive_loss(
    logits, exact_component_target, negative_annulus_target, component_valid,
):
    exact_target = exact_component_target.bool()
    negative_target = negative_annulus_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    positive_losses = []
    negative_losses = []
    rank_losses = []
    positive_logits = []
    hard_negative_logits = []
    selected_negative_counts = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]].float()
        negative = logits[batch_index][negative_target[batch_index]].float()
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError(
                'A valid r7 component must have positive voxels and a '
                'lesion-free negative annulus.'
            )
        hard_count = min(
            int(negative.numel()),
            max(
                R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * int(positive.numel()),
            ),
        )
        hard_negative = torch.topk(
            negative, k=hard_count, largest=True
        ).values
        positive_loss = F.softplus(-positive).mean()
        negative_loss = F.softplus(hard_negative).mean()
        rank_loss = F.softplus(
            R7_LOGIT_MARGIN - positive.mean() + hard_negative.mean()
        )
        contrastive_loss = (
            R7_POSITIVE_WEIGHT * positive_loss
            + R7_HARD_NEGATIVE_WEIGHT * negative_loss
            + R7_RANK_WEIGHT * rank_loss
        )
        losses.append(contrastive_loss)
        positive_losses.append(positive_loss)
        negative_losses.append(negative_loss)
        rank_losses.append(rank_loss)
        positive_logits.append(positive.mean())
        hard_negative_logits.append(hard_negative.mean())
        selected_negative_counts.append(hard_count)

    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'local_contrastive_loss': zero,
            'component_positive_loss': zero,
            'component_hard_negative_loss': zero,
            'component_rank_loss': zero,
            'mean_component_positive_logit': zero.detach(),
            'mean_component_hard_negative_logit': zero.detach(),
            'selected_hard_negative_voxels': zero.detach(),
            'selected_components_in_batch': zero.detach(),
        }
    return torch.stack(losses).mean(), {
        'local_contrastive_loss': torch.stack(losses).mean(),
        'component_positive_loss': torch.stack(positive_losses).mean(),
        'component_hard_negative_loss': torch.stack(negative_losses).mean(),
        'component_rank_loss': torch.stack(rank_losses).mean(),
        'mean_component_positive_logit': torch.stack(positive_logits).mean(),
        'mean_component_hard_negative_logit': torch.stack(
            hard_negative_logits
        ).mean(),
        'selected_hard_negative_voxels': torch.as_tensor(
            float(sum(selected_negative_counts)), device=logits.device
        ),
        'selected_components_in_batch': torch.as_tensor(
            float(len(losses)), device=logits.device
        ),
    }


def r7_local_contrastive_proposal_loss(
    logits, full_target, exact_component_target,
    negative_annulus_target, component_valid, epoch,
):
    whole_dice = pilot_soft_dice_loss(logits, full_target).mean()
    whole_bce = pilot_focal_bce_loss(logits, full_target).mean()
    whole_loss = 0.65 * whole_dice + 0.35 * whole_bce
    contrastive_loss, contrastive_parts = r7_local_contrastive_loss(
        logits, exact_component_target, negative_annulus_target,
        component_valid,
    )
    coefficient = r7_component_coefficient(epoch)
    proposal_loss = whole_loss + coefficient * contrastive_loss
    return proposal_loss, {
        'proposal_whole_mask_loss': whole_loss,
        'proposal_dice_loss': whole_dice,
        'proposal_focal_bce_loss': whole_bce,
        **contrastive_parts,
        'component_loss_coefficient': torch.as_tensor(
            coefficient, device=logits.device, dtype=torch.float32
        ),
    }


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(
    model, cpu_batch, device, epoch, batch_index,
    dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    if dropout_flags_override is not None:
        required_flags = {
            'force_atlas_dropout', 'force_vascular_dropout',
            'force_occlusion_dropout', 'force_clinical_dropout',
            'force_global_dropout', 'force_arterial_dropout',
        }
        if set(dropout_flags_override) != required_flags:
            raise RuntimeError(
                'Dropout override keys do not match the frozen branch set.'
            )
        flags = {
            name: bool(dropout_flags_override[name])
            for name in sorted(required_flags)
        }
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_loss, proposal_parts = r7_local_contrastive_proposal_loss(
        output['global_proposal_logits'], global_target,
        batch['selected_component_target'],
        batch['selected_component_negative_annulus'],
        batch['selected_component_valid'], epoch,
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        **proposal_parts,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'arterial_valid_nodes': arterial_valid.sum().detach().float(),
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'arterial_dropped': torch.as_tensor(float(flags['force_arterial_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_volume_bounded_prediction(probability, target_voxels):
    probability = np.asarray(probability, dtype=np.float32)
    budget = min(
        probability.size,
        max(1, int(math.ceil(
            R7_VOLUME_BUDGET_MULTIPLIER * int(target_voxels)
        ))),
    )
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape), budget


def pilot_histogram_average_precision(positive_histogram, negative_histogram):
    positive = np.asarray(positive_histogram, dtype=np.float64)[::-1]
    negative = np.asarray(negative_histogram, dtype=np.float64)[::-1]
    total_positive = float(positive.sum())
    if total_positive <= 0:
        return float('nan')
    cumulative_positive = np.cumsum(positive)
    cumulative_total = np.cumsum(positive + negative)
    precision = cumulative_positive / np.maximum(cumulative_total, 1.0)
    return float((precision * positive).sum() / total_positive)


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    voxel_precision = total_intersection / max(total_prediction, 1)
    predicted_to_target_volume_ratio = (
        total_prediction / max(total_target, 1)
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'voxel_precision': float(voxel_precision),
        'predicted_to_target_volume_ratio': float(
            predicted_to_target_volume_ratio
        ),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['volume_calibration_score'] = np.exp(-np.abs(
        np.log(np.maximum(
            proposal_threshold_df['predicted_to_target_volume_ratio'], 1e-6
        ))
    ))
    proposal_threshold_df['proposal_selection_score'] = (
        0.35 * proposal_threshold_df['positive_global_dice']
        + 0.15 * proposal_threshold_df['lesion_f1']
        + 0.20 * proposal_threshold_df['voxel_precision']
        + 0.15 * proposal_threshold_df['component_recall']
        + 0.15 * proposal_threshold_df['volume_calibration_score']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        [
            'proposal_selection_score', 'positive_global_dice',
            'voxel_precision', 'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    proposal_ap_positive_histogram = np.zeros(256, dtype=np.int64)
    proposal_ap_negative_histogram = np.zeros(256, dtype=np.int64)
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        proposal_intersection = int((proposal_prediction & target).sum())
        proposal_predicted_voxels = int(proposal_prediction.sum())
        proposal_distance = None
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        budgeted_proposal_prediction = None
        budgeted_proposal_distance = None
        proposal_volume_budget_voxels = 0
        if hard_eligible and target_voxels > 0:
            (
                budgeted_proposal_prediction,
                proposal_volume_budget_voxels,
            ) = pilot_volume_bounded_prediction(proposal, target_voxels)
            budgeted_proposal_distance = ndimage.distance_transform_edt(
                ~budgeted_proposal_prediction, sampling=case['spacing']
            )
            proposal_ap_positive_histogram += np.histogram(
                proposal[target], bins=256, range=(0.0, 1.0)
            )[0]
            proposal_ap_negative_histogram += np.histogram(
                proposal[~target], bins=256, range=(0.0, 1.0)
            )[0]
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
            'proposal_predicted_voxels': proposal_predicted_voxels,
            'proposal_intersection_voxels': proposal_intersection,
            'proposal_voxel_precision': (
                proposal_intersection / max(proposal_predicted_voxels, 1)
            ),
            'proposal_voxel_recall': (
                proposal_intersection / max(target_voxels, 1)
            ),
            'proposal_predicted_to_target_volume_ratio': (
                proposal_predicted_voxels / max(target_voxels, 1)
            ),
            'proposal_volume_budget_voxels': proposal_volume_budget_voxels,
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            budgeted_proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    budgeted_proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                budgeted_proposal_max_iou = float(
                    budgeted_proposal_iou[component_id][
                        'maximum_component_iou'
                    ]
                )
                proposal_distance_mm = float('nan')
                if proposal_distance is not None:
                    proposal_distance_mm = float(
                        proposal_distance[component_slice][component_local].min(
                            initial=np.inf
                        )
                    )
                budgeted_proposal_distance_mm = float(
                    budgeted_proposal_distance[component_slice][
                        component_local
                    ].min(initial=np.inf)
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                budgeted_proposal_covered = (
                    budgeted_proposal_max_iou >= 0.05
                )
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += int(budgeted_proposal_covered)
                tier_counts[tier][3] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'proposal_minimum_distance_mm': proposal_distance_mm,
                    'budgeted_proposal_maximum_component_iou': (
                        budgeted_proposal_max_iou
                    ),
                    'budgeted_proposal_minimum_distance_mm': (
                        budgeted_proposal_distance_mm
                    ),
                    'proposal_within_2mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 2.0
                    ),
                    'proposal_within_4mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 4.0
                    ),
                    'proposal_within_8mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_within_2mm': bool(
                        budgeted_proposal_distance_mm <= 2.0
                    ),
                    'budgeted_proposal_within_4mm': bool(
                        budgeted_proposal_distance_mm <= 4.0
                    ),
                    'budgeted_proposal_within_8mm': bool(
                        budgeted_proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_covered': bool(
                        budgeted_proposal_covered
                    ),
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, budgeted_proposal_tp, total = tier_counts[tier]
        tier_components = [
            item for item in component_rows
            if item['component_tier'] == tier
        ]
        finite_distances = [
            float(item['proposal_minimum_distance_mm'])
            for item in tier_components
            if np.isfinite(item['proposal_minimum_distance_mm'])
        ]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
            'budgeted_proposal_component_recall': (
                budgeted_proposal_tp / max(total, 1)
            ),
            'proposal_within_2mm_fraction': float(np.mean([
                item['proposal_within_2mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_4mm_fraction': float(np.mean([
                item['proposal_within_4mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_8mm_fraction': float(np.mean([
                item['proposal_within_8mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_2mm_fraction': float(np.mean([
                item['budgeted_proposal_within_2mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_4mm_fraction': float(np.mean([
                item['budgeted_proposal_within_4mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_8mm_fraction': float(np.mean([
                item['budgeted_proposal_within_8mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'median_finite_proposal_distance_mm': float(
                np.median(finite_distances)
            ) if finite_distances else float('nan'),
            'mean_proposal_maximum_probability': float(np.mean([
                item['proposal_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'mean_final_maximum_probability': float(np.mean([
                item['final_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'representation_failure_fraction': float(np.mean([
                item['representation_failure_below_0p10']
                for item in tier_components
            ])) if tier_components else 0.0,
        })
    small_component_rows = [
        item for item in component_rows
        if item['component_tier'] in set(COMPONENT_TIERS[:4])
    ]
    proposal_voxel_average_precision = pilot_histogram_average_precision(
        proposal_ap_positive_histogram, proposal_ap_negative_histogram
    )
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'proposal_voxel_precision': float(
            proposal_selected['voxel_precision']
        ),
        'proposal_predicted_to_target_volume_ratio': float(
            proposal_selected['predicted_to_target_volume_ratio']
        ),
        'proposal_volume_calibration_score': float(
            proposal_selected['volume_calibration_score']
        ),
        'proposal_voxel_average_precision_256bin': float(
            proposal_voxel_average_precision
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'small_components_audited': len(small_component_rows),
        'small_component_proposal_within_2mm_fraction': float(np.mean([
            item['proposal_within_2mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_4mm_fraction': float(np.mean([
            item['proposal_within_4mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_8mm_fraction': float(np.mean([
            item['proposal_within_8mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_budgeted_proposal_within_2mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_2mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_4mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_4mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_8mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_8mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered'] for item in component_rows
        ])) if component_rows else 0.0,
        'small_budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_representation_failure_fraction': float(np.mean([
            item['representation_failure_below_0p10']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    required_finite_summary_metrics = [
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'proposal_global_dice', 'proposal_component_recall',
        'proposal_voxel_precision',
        'proposal_predicted_to_target_volume_ratio',
        'proposal_voxel_average_precision_256bin',
        'budgeted_proposal_component_recall',
        'small_budgeted_proposal_component_recall',
        'small_component_budgeted_proposal_within_2mm_fraction',
        'small_component_budgeted_proposal_within_4mm_fraction',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'small_component_representation_failure_fraction',
        'pilot_checkpoint_score',
    ]
    nonfinite_summary_metrics = [
        name for name in required_finite_summary_metrics
        if not np.isfinite(float(summary[name]))
    ]
    if nonfinite_summary_metrics:
        raise RuntimeError(
            'Validation produced non-finite required metrics: '
            + ', '.join(nonfinite_summary_metrics)
        )
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
            and completed.get('best_checkpoint_sha256')
            == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R7 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_index = None
    smoke_subject_id = None
    for candidate_index, entry in enumerate(dataset.schedule):
        if entry['sample_type'] != 'positive':
            continue
        candidate_subject = str(entry['subject_id'])
        if not as_bool(
            fit_row_by_subject[candidate_subject]['trial8_atlas_enabled']
        ):
            continue
        candidate_sample = dataset[candidate_index]
        candidate_checks = {
            'atlas_enabled': bool(np.asarray(
                candidate_sample['atlas_enabled']
            ).reshape(-1)[0]),
            'territory_nodes_valid': bool(np.asarray(
                candidate_sample['territory_valid']
            ).astype(bool).any()),
            'vessel_nodes_valid': bool(np.asarray(
                candidate_sample['vessel_valid']
            ).astype(bool).any()),
            'selected_component_valid': bool(np.asarray(
                candidate_sample['selected_component_valid']
            ).reshape(-1)[0]),
        }
        if all(candidate_checks.values()):
            smoke_index = int(candidate_index)
            smoke_subject_id = candidate_subject
            break
    if smoke_index is None:
        raise RuntimeError(
            'No positive fold-0-fit sample has simultaneous atlas, territory, '
            'vessel, and selected-component support for the all-branch smoke '
            'test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    smoke_dropout_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device,
            epoch=R7_COMPONENT_LOSS_FULL_EPOCH, batch_index=0,
            dropout_flags_override=smoke_dropout_flags,
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    r7_smoke_checks = {
        'positive_sample_selected': bool(
            smoke_batch['selected_component_valid'].bool().all().item()
        ),
        'atlas_enabled_for_smoke': bool(
            smoke_batch['atlas_enabled'].bool().all().item()
        ),
        'territory_support_for_smoke': bool(
            smoke_batch['territory_valid'].bool().any().item()
        ),
        'vessel_support_for_smoke': bool(
            smoke_batch['vessel_valid'].bool().any().item()
        ),
        'projected_arterial_nodes_valid_for_smoke': float(
            smoke_parts['arterial_valid_nodes'].detach().float().cpu()
        ) > 0.0,
        'arterial_loss_finite_for_smoke': bool(torch.isfinite(
            smoke_parts['arterial_loss']
        ).item()),
        'all_branch_dropouts_disabled_for_smoke': not any(
            smoke_dropout_flags.values()
        ),
        'exact_component_target_nonempty': bool(
            smoke_batch['selected_component_target'].bool().any().item()
        ),
        'negative_annulus_nonempty': bool(
            smoke_batch['selected_component_negative_annulus']
            .bool().any().item()
        ),
        'negative_annulus_disjoint_from_exact_component': not bool(torch.any(
            smoke_batch['selected_component_target'].bool()
            & smoke_batch['selected_component_negative_annulus'].bool()
        ).item()),
        'local_contrastive_loss_positive': float(
            smoke_parts['local_contrastive_loss']
            .detach().float().cpu()
        ) > 0.0,
        'smoke_uses_maximum_contrastive_coefficient': math.isclose(
            float(smoke_parts['component_loss_coefficient']
                  .detach().float().cpu()),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT,
            rel_tol=0.0, abs_tol=1e-7,
        ),
        'whole_mask_proposal_loss_finite': bool(torch.isfinite(
            smoke_parts['proposal_whole_mask_loss']
        ).item()),
    }
    if not all(r7_smoke_checks.values()):
        failed = [
            name for name, passed in r7_smoke_checks.items() if not passed
        ]
        raise RuntimeError(
            'R7 local-contrastive smoke checks failed: ' + ', '.join(failed)
        )
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    if R7_R5_BASELINE_DIAGNOSTIC_PATH.exists():
        r5_baseline_diagnostic = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
        if (
            r5_baseline_diagnostic.get('r5_checkpoint_sha256')
            != r5_checkpoint_sha256
            or r5_baseline_diagnostic.get('metric_contract_revision')
            != PILOT_CODE_REVISION
        ):
            raise RuntimeError(
                'Existing r5 volume-bounded baseline is incompatible with '
                'the current r7 contract.'
            )
        r5_recomputed_summary = dict(
            r5_baseline_diagnostic['validation_summary']
        )
    else:
        rng_state_before_baseline = pilot_capture_rng_state()
        r5_checkpoint = torch.load(
            R5_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
        )
        r5_checkpoint_checks = {
            'pilot_version': (
                r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
            ),
            'code_revision': (
                r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
            ),
            'parameter_count': int(
                r5_checkpoint.get('parameter_count', -1)
            ) == EXPECTED_V22_PARAMETERS,
        }
        if not all(r5_checkpoint_checks.values()):
            failed = [
                name for name, passed in r5_checkpoint_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Unsafe r5 read-only baseline checkpoint: '
                + ', '.join(failed)
            )
        r5_baseline_model = VascuFateNetV22(pilot_schema).to(device).eval()
        r5_baseline_model.load_state_dict(r5_checkpoint['ema_model_state'])
        for parameter in r5_baseline_model.parameters():
            parameter.requires_grad_(False)
        print(
            '\nRecomputing the completed r5 best checkpoint with the r7 '
            'precision- and volume-bounded diagnostic contract.'
        )
        r5_recomputed_summary, r5_recomputed_tiers = pilot_validate(
            r5_baseline_model, epoch=0, device=device
        )
        r5_baseline_diagnostic = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'metric_contract_revision': PILOT_CODE_REVISION,
            'r5_pilot_version': R5_PILOT_VERSION,
            'r5_code_revision': R5_PILOT_CODE_REVISION,
            'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'validation_summary': r5_recomputed_summary,
            'component_tier_metrics': r5_recomputed_tiers,
            'r5_checkpoint_loaded_into_r7_model': False,
            'optimizer_steps': 0,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        }
        atomic_write_json(
            r5_baseline_diagnostic, R7_R5_BASELINE_DIAGNOSTIC_PATH
        )
        del r5_baseline_model, r5_checkpoint
        gc.collect()
        torch.cuda.empty_cache()
        pilot_restore_rng_state(rng_state_before_baseline)

    
    
    
    
    del model, ema_model
    gc.collect()
    torch.cuda.empty_cache()
    set_determinism(PILOT_SEED)
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed during lineage reset.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    lineage_smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True
    )
    lineage_smoke_batch = next(iter(lineage_smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        lineage_smoke_loss, lineage_smoke_parts = (
            pilot_compute_training_loss(
                model, lineage_smoke_batch, device, epoch=1, batch_index=0
            )
        )
    if not bool(torch.isfinite(lineage_smoke_loss).item()):
        raise RuntimeError('R7 controlled-lineage smoke loss is non-finite.')
    if float(
        lineage_smoke_parts['component_loss_coefficient']
        .detach().float().cpu()
    ) != 0.0:
        raise RuntimeError(
            'R7 controlled-lineage epoch-1 coefficient is not exactly zero.'
        )
    lineage_smoke_loss.backward()
    lineage_gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            parameter.grad
            for name, parameter in model.named_parameters()
            if name.startswith(prefix) and parameter.grad is not None
        ]
        lineage_gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(gradient).all().item())
            for gradient in gradients
        ) and any(
            float(gradient.detach().float().norm().cpu()) > 0.0
            for gradient in gradients
        )
    model.zero_grad(set_to_none=True)
    if not all(lineage_gradient_checks.values()):
        failed = [
            name for name, passed in lineage_gradient_checks.items()
            if not passed
        ]
        raise RuntimeError(
            'R7 controlled-lineage gradient smoke failed: '
            + ', '.join(failed)
        )

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r7_last_checkpoint_only',
        'strict_best_checkpoint_policy': (
            'save every strict numeric pilot-score improvement'
        ),
        'minimum_delta_policy': (
            'used only to reset early-stopping patience'
        ),
        'r7_preflight_gate_sha256': file_sha256(R7_PREFLIGHT_GATE_PATH),
        'r7_proposal_loss_policy_sha256': file_sha256(R7_POLICY_PATH),
        'component_loss_start_epoch': R7_COMPONENT_LOSS_START_EPOCH,
        'component_loss_full_epoch': R7_COMPONENT_LOSS_FULL_EPOCH,
        'component_loss_maximum_coefficient': (
            R7_COMPONENT_LOSS_MAX_COEFFICIENT
        ),
        'component_positive_weight': R7_POSITIVE_WEIGHT,
        'component_hard_negative_weight': R7_HARD_NEGATIVE_WEIGHT,
        'component_rank_weight': R7_RANK_WEIGHT,
        'component_inner_radius_mm': R7_INNER_RADIUS_MM,
        'component_outer_radius_mm': R7_OUTER_RADIUS_MM,
        'component_logit_margin': R7_LOGIT_MARGIN,
        'component_eligible_tiers': sorted(R7_SMALL_COMPONENT_TIERS),
        'component_coefficient_schedule': {
            str(epoch): coefficient
            for epoch, coefficient in R7_COMPONENT_COEFFICIENT_SCHEDULE.items()
        },
        'proposal_volume_budget_multiplier': R7_VOLUME_BUDGET_MULTIPLIER,
        'proposal_volume_budget_policy': (
            'diagnostic only: retain the top 3x ground-truth target voxels; '
            'never used as an inference rule or checkpoint-selection input'
        ),
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'controlled_lineage_matches_r5_seed': True,
        'controlled_lineage_epoch1_component_coefficient': 0.0,
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'controlled_lineage_gradient_checks': lineage_gradient_checks,
        'controlled_lineage_smoke_loss': float(
            lineage_smoke_loss.detach().cpu()
        ),
        'r7_selected_component_smoke_checks': r7_smoke_checks,
        'r7_component_coefficient_schedule_verified': True,
        'r7_smoke_subject_id': smoke_subject_id,
        'r7_smoke_schedule_index': smoke_index,
        'r7_smoke_dropout_flags': smoke_dropout_flags,
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R7 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R7 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r7 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Controlled r5-seed lineage:          verified')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('Local contrastive rescue:              small tiers, epochs 8-40')
    print('Maximum contrastive coefficient:       0.10')
    print('Proposal anti-shortcut diagnostics:    oracle-volume-bounded (audit only)')
    print('R5 recomputed mean/global Dice:        '
          f'{r5_recomputed_summary["positive_mean_dice"]:.4f} / '
          f'{r5_recomputed_summary["positive_global_dice"]:.4f}')
    print('R5 oracle-budgeted small within 8 mm:  '
          f'{r5_recomputed_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}')
    print('Best-checkpoint minimum delta:        disabled (strict numeric best)')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        strict_best_improved = current_score > best_score
        patience_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if strict_best_improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
        if patience_improved:
            patience_reference_score = current_score
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        epoch_tier_metrics = {
            str(row['component_tier']): row for row in tier_rows
        }
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'proposal_voxel_precision': summary['proposal_voxel_precision'],
            'proposal_predicted_to_target_volume_ratio': summary[
                'proposal_predicted_to_target_volume_ratio'
            ],
            'proposal_voxel_average_precision_256bin': summary[
                'proposal_voxel_average_precision_256bin'
            ],
            'budgeted_proposal_component_recall': summary[
                'budgeted_proposal_component_recall'
            ],
            'small_component_proposal_within_2mm_fraction': summary[
                'small_component_proposal_within_2mm_fraction'
            ],
            'small_component_proposal_within_4mm_fraction': summary[
                'small_component_proposal_within_4mm_fraction'
            ],
            'small_component_proposal_within_8mm_fraction': summary[
                'small_component_proposal_within_8mm_fraction'
            ],
            'small_component_budgeted_proposal_within_8mm_fraction': summary[
                'small_component_budgeted_proposal_within_8mm_fraction'
            ],
            'small_component_representation_failure_fraction': summary[
                'small_component_representation_failure_fraction'
            ],
            **{
                f'{tier_name}_{metric_name}': float(
                    epoch_tier_metrics.get(tier_name, {}).get(
                        metric_name, float('nan')
                    )
                )
                for tier_name in COMPONENT_TIERS
                for metric_name in (
                    'final_component_recall',
                    'proposal_component_recall',
                    'budgeted_proposal_component_recall',
                    'proposal_within_8mm_fraction',
                    'budgeted_proposal_within_8mm_fraction',
                    'representation_failure_fraction',
                )
            },
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'strict_best_improved': strict_best_improved,
            'patience_improved_by_minimum_delta': patience_improved,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if strict_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'contrastive={running["local_contrastive_loss"]/max(processed,1):.4f}, '
            f'coef={running["component_loss_coefficient"]/max(processed,1):.3f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        print(
            '  R7 raw proposal geometry: small within 2/4/8 mm='
            f'{summary["small_component_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_8mm_fraction"]:.4f}, '
            'small representation failure='
            f'{summary["small_component_representation_failure_fraction"]:.4f}'
        )
        print(
            '  Oracle-volume-bounded diagnostic: small within 2/4/8 mm='
            f'{summary["small_component_budgeted_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}, '
            f'component recall={summary["budgeted_proposal_component_recall"]:.4f}, '
            f'precision/AP={summary["proposal_voxel_precision"]:.4f}/'
            f'{summary["proposal_voxel_average_precision_256bin"]:.4f}, '
            f'volume ratio={summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_baseline = r7_policy['r5_baseline']
    success_policy = r7_policy['r7_directional_success_gates']
    best_tier_by_name = {
        str(row['component_tier']): row for row in best_tier_rows
    }
    best_large_proposal_recall = float(
        best_tier_by_name.get('large', {}).get(
            'proposal_component_recall', 0.0
        )
    )
    mean_dice_improvement = (
        float(best_summary['positive_mean_dice'])
        - float(r5_recomputed_summary['positive_mean_dice'])
    )
    budgeted_small_within_8mm_improvement = (
        float(best_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
        - float(r5_recomputed_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
    )
    small_representation_failure_reduction = (
        float(r5_recomputed_summary[
            'small_component_representation_failure_fraction'
        ])
        - float(best_summary[
            'small_component_representation_failure_fraction'
        ])
    )
    global_dice_regression = (
        float(r5_recomputed_summary['positive_global_dice'])
        - float(best_summary['positive_global_dice'])
    )
    proposal_dice_regression = (
        float(r5_recomputed_summary['proposal_global_dice'])
        - float(best_summary['proposal_global_dice'])
    )
    r7_success_checks = {
        'mean_dice_does_not_materially_regress': (
            -mean_dice_improvement
            <= float(success_policy[
                'mean_dice_maximum_allowed_regression'
            ])
        ),
        'mean_dice_reaches_target_improvement': (
            mean_dice_improvement
            >= float(success_policy[
                'mean_dice_target_absolute_improvement'
            ])
        ),
        'volume_bounded_small_within_8mm_improves': (
            budgeted_small_within_8mm_improvement
            >= float(success_policy[
                'volume_bounded_small_within_8mm_absolute_improvement_at_least'
            ])
        ),
        'small_representation_failure_reduced_by_required_margin': (
            small_representation_failure_reduction
            >= float(success_policy[
                'small_representation_failure_absolute_reduction_at_least'
            ])
        ),
        'proposal_component_recall_meets_minimum': (
            float(best_summary['proposal_component_recall'])
            >= float(success_policy['proposal_component_recall_minimum'])
        ),
        'global_dice_regression_within_limit': (
            global_dice_regression
            <= float(success_policy[
                'global_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_dice_regression_within_limit': (
            proposal_dice_regression
            <= float(success_policy[
                'proposal_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_volume_ratio_is_bounded': (
            float(best_summary[
                'proposal_predicted_to_target_volume_ratio'
            ])
            <= float(success_policy[
                'proposal_predicted_to_target_volume_ratio_maximum'
            ])
        ),
        'empty_false_positive_volume_within_limit': (
            float(best_summary['empty_false_positive_volume_ml'])
            <= float(success_policy[
                'mean_false_positive_volume_ml_maximum'
            ])
        ),
        'large_component_proposal_recall_meets_minimum': (
            best_large_proposal_recall
            >= float(success_policy[
                'large_component_proposal_recall_minimum'
            ])
        ),
    }
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'r5_baseline_evidence': r5_baseline,
        'r5_recomputed_under_r7_metric_contract': r5_recomputed_summary,
        'r5_read_only_baseline_diagnostic_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r7_directional_success_policy': success_policy,
        'r7_directional_success_checks': r7_success_checks,
        'r7_directional_success_passed': all(r7_success_checks.values()),
        'r7_directional_effects': {
            'mean_dice_absolute_change': mean_dice_improvement,
            'volume_bounded_small_within_8mm_absolute_change': (
                budgeted_small_within_8mm_improvement
            ),
            'small_representation_failure_absolute_reduction': (
                small_representation_failure_reduction
            ),
            'global_dice_regression': global_dice_regression,
            'proposal_dice_regression': proposal_dice_regression,
            'proposal_predicted_to_target_volume_ratio': float(
                best_summary['proposal_predicted_to_target_volume_ratio']
            ),
            'proposal_voxel_average_precision_256bin': float(
                best_summary['proposal_voxel_average_precision_256bin']
            ),
            'large_component_proposal_recall': (
                best_large_proposal_recall
            ),
        },
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(
        'Raw small proposal within 8 mm:   '
        f'{best_summary["small_component_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Oracle-budgeted small within 8 mm: '
        f'{best_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Proposal precision / AP:          '
        f'{best_summary["proposal_voxel_precision"]:.4f} / '
        f'{best_summary["proposal_voxel_average_precision_256bin"]:.4f}'
    )
    print(
        'Proposal predicted/target ratio:  '
        f'{best_summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
    )
    print(
        'Mean Dice change vs recomputed r5:'
        f' {mean_dice_improvement:+.4f}'
    )
    print(
        'Budgeted within-8-mm change vs r5:'
        f' {budgeted_small_within_8mm_improvement:+.4f}'
    )
    print(
        'Small representation failure:     '
        f'{best_summary["small_component_representation_failure_fraction"]:.4f}'
    )
    print(
        'R7 directional success gates:     '
        f'{all(r7_success_checks.values())}'
    )
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')






R7D_EXPOSURE_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure.csv'
R7D_EXPOSURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure_by_tier.csv'
)
R7D_RESOLUTION_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_component_audit.csv'
)
R7D_RESOLUTION_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_summary.csv'
)
R7D_PAIRED_CASE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_case_metrics.csv'
)
R7D_PAIRED_COMPONENT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_component_metrics.csv'
)
R7D_MARGIN_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_train_heldout_margin_audit.csv'
)
R7D_MARGIN_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_margin_summary.csv'
)
R7D_VOLUME_BUDGET_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_audit.csv'
)
R7D_VOLUME_BUDGET_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_summary.csv'
)
R7D_CHECKPOINT_PARETO_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'checkpoint_metric_pareto.csv'
)
R7D_BRANCH_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'branch_reliance_comparison.csv'
)
R7D_VISUAL_CANDIDATES_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'visual_qa_candidates.csv'
)
R7D_CHECKS_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'diagnostic_checks.csv'
R7D_TARGET_ACCESS_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'development_target_access_audit.csv'
)

R7D_OUTPUT_PATHS = [
    R7D_EXPOSURE_PATH,
    R7D_EXPOSURE_SUMMARY_PATH,
    R7D_RESOLUTION_PATH,
    R7D_RESOLUTION_SUMMARY_PATH,
    R7D_PAIRED_CASE_PATH,
    R7D_PAIRED_COMPONENT_PATH,
    R7D_MARGIN_PATH,
    R7D_MARGIN_SUMMARY_PATH,
    R7D_VOLUME_BUDGET_PATH,
    R7D_VOLUME_BUDGET_SUMMARY_PATH,
    R7D_CHECKPOINT_PARETO_PATH,
    R7D_BRANCH_PATH,
    R7D_VISUAL_CANDIDATES_PATH,
    R7D_CHECKS_PATH,
    R7D_TARGET_ACCESS_PATH,
    R7_FAILURE_SUMMARY_PATH,
]


def r7d_existing_result_is_complete():
    if not R7_FAILURE_GATE_PATH.exists():
        return False
    gate = load_json(R7_FAILURE_GATE_PATH)
    if not (
        gate.get('diagnostic_complete') is True
        and gate.get('diagnostic_version') == R7_FAILURE_DIAGNOSTIC_VERSION
        and gate.get('code_revision') == R7_FAILURE_DIAGNOSTIC_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_checkpoint_sha256')
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    ):
        return False
    output_hashes = gate.get('output_sha256', {})
    return all(
        path.exists()
        and output_hashes.get(path.name) == file_sha256(path)
        for path in R7D_OUTPUT_PATHS
    )


def r7d_gini(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0 or float(values.sum()) <= 0.0:
        return 0.0
    values = np.sort(values)
    index = np.arange(1, values.size + 1, dtype=np.float64)
    return float(
        np.sum((2.0 * index - values.size - 1.0) * values)
        / (values.size * values.sum())
    )


def r7d_component_support(row):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    mode = str(row['trial8_supervision_mode'])
    if mode == 'hard_and_soft':
        support = binary
    elif mode == 'soft_only_subvoxel':
        support = soft > 0
    elif mode == 'true_empty':
        support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unsupported supervision mode {mode!r}.'
        )
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: non-finite target arrays.')
    return binary, soft, support, spacing


def r7d_component_schedule_exposure():
    counts = defaultdict(int)
    first_epoch = {}
    last_epoch = {}
    for epoch in range(R7_COMPONENT_LOSS_START_EPOCH, MAX_EPOCHS + 1):
        schedule = pilot_build_epoch_schedule(epoch)
        for entry in schedule:
            if (
                str(entry['sample_type']) != 'positive'
                or str(entry['component_tier']) not in R7_SMALL_COMPONENT_TIERS
            ):
                continue
            key = (
                str(entry['subject_id']), int(entry['atomic_component_id'])
            )
            counts[key] += 1
            first_epoch.setdefault(key, int(epoch))
            last_epoch[key] = int(epoch)

    rows = []
    for item in component_df.sort_values(
        ['subject_id', 'atomic_component_id']
    ).to_dict('records'):
        key = (str(item['subject_id']), int(item['atomic_component_id']))
        tier = str(item['component_tier'])
        eligible = tier in R7_SMALL_COMPONENT_TIERS
        selections = int(counts.get(key, 0)) if eligible else 0
        rows.append({
            'subject_id': key[0],
            'atomic_component_id': key[1],
            'component_tier': tier,
            'component_voxels': int(item['component_voxels']),
            'component_volume_ml': float(item['component_volume_ml']),
            'global_proposal_voxels': int(item['global_proposal_voxels']),
            'local_contrastive_eligible': bool(eligible),
            'epochs_eligible': (
                MAX_EPOCHS - R7_COMPONENT_LOSS_START_EPOCH + 1
                if eligible else 0
            ),
            'selections_epochs_8_to_40': selections,
            'first_selected_epoch': first_epoch.get(key, np.nan),
            'last_selected_epoch': last_epoch.get(key, np.nan),
            'never_selected': bool(eligible and selections == 0),
            'selected_once': bool(eligible and selections == 1),
            'selected_multiple_times': bool(eligible and selections > 1),
        })
    frame = pd.DataFrame(rows)
    summary_rows = []
    for tier in COMPONENT_TIERS:
        group = frame[frame['component_tier'].astype(str) == tier]
        eligible = group[group['local_contrastive_eligible'].astype(bool)]
        values = eligible['selections_epochs_8_to_40'].to_numpy(
            dtype=np.float64
        )
        summary_rows.append({
            'component_tier': tier,
            'components': int(len(group)),
            'eligible_components': int(len(eligible)),
            'never_selected_components': int(
                eligible['never_selected'].sum()
            ),
            'never_selected_fraction': float(
                eligible['never_selected'].mean()
            ) if len(eligible) else 0.0,
            'selected_once_fraction': float(
                eligible['selected_once'].mean()
            ) if len(eligible) else 0.0,
            'minimum_selections': int(values.min()) if len(values) else 0,
            'median_selections': float(np.median(values)) if len(values) else 0.0,
            'mean_selections': float(values.mean()) if len(values) else 0.0,
            'maximum_selections': int(values.max()) if len(values) else 0,
            'selection_gini': r7d_gini(values),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(frame, R7D_EXPOSURE_PATH)
    atomic_write_csv(summary, R7D_EXPOSURE_SUMMARY_PATH)
    return frame, summary


def r7d_maximum_component_cell_matching(cell_sets):
    matched_cell = {}

    def visit(component_index, seen):
        for cell_id in sorted(cell_sets[component_index]):
            if cell_id in seen:
                continue
            seen.add(cell_id)
            if (
                cell_id not in matched_cell
                or visit(matched_cell[cell_id], seen)
            ):
                matched_cell[cell_id] = component_index
                return True
        return False

    matched = 0
    for component_index in range(len(cell_sets)):
        matched += int(visit(component_index, set()))
    return int(matched)


def r7d_resolution_audit():
    component_rows = []
    subject_rows = []
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    canvas_shape = np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    global_shape = tuple(int(v) for v in GLOBAL_INPUT_SHAPE)
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            if partition == 'fold0_fit':
                expected = len(component_records.get(subject_id, []))
                if int(count) != int(expected):
                    raise RuntimeError(
                        f'{subject_id}: resolution audit reconstructed '
                        f'{count} components; frozen audit has {expected}.'
                    )
            pad_start = (canvas_shape - np.asarray(support.shape)) // 2
            cell_sets = []
            component_metadata = []
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for component_id in range(1, int(count) + 1):
                coordinates = np.argwhere(labels == component_id)
                pooled = np.unique(
                    (coordinates + pad_start) // downsample, axis=0
                )
                cell_ids = set(
                    np.ravel_multi_index(pooled.T, global_shape).tolist()
                )
                cell_sets.append(cell_ids)
                voxels = int(len(coordinates))
                volume_ml = voxels * voxel_volume_ml
                component_metadata.append((voxels, volume_ml, pooled))
            cell_frequency = defaultdict(int)
            signature_frequency = defaultdict(int)
            for cells in cell_sets:
                for cell_id in cells:
                    cell_frequency[cell_id] += 1
                signature_frequency[tuple(sorted(cells))] += 1
            matched = r7d_maximum_component_cell_matching(cell_sets)
            collision_components = 0
            duplicate_signature_components = 0
            for component_index, cells in enumerate(cell_sets):
                voxels, volume_ml, pooled = component_metadata[component_index]
                shared_cells = sum(
                    int(cell_frequency[cell_id] > 1) for cell_id in cells
                )
                duplicate_signature = (
                    signature_frequency[tuple(sorted(cells))] > 1
                )
                collision = shared_cells > 0
                collision_components += int(collision)
                duplicate_signature_components += int(duplicate_signature)
                component_rows.append({
                    'development_partition': partition,
                    'subject_id': subject_id,
                    'component_id': int(component_index + 1),
                    'component_tier': pilot_component_tier(volume_ml, voxels),
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'global_proposal_voxels': int(len(cells)),
                    'single_global_voxel': bool(len(cells) == 1),
                    'shared_global_voxels': int(shared_cells),
                    'shares_any_global_voxel': bool(collision),
                    'duplicate_pooled_signature': bool(duplicate_signature),
                    'native_voxels_per_global_voxel': float(
                        voxels / max(len(cells), 1)
                    ),
                })
            subject_rows.append({
                'development_partition': partition,
                'subject_id': subject_id,
                'components': int(count),
                'distinct_global_voxels_used': int(len(cell_frequency)),
                'components_sharing_any_global_voxel': int(collision_components),
                'components_with_duplicate_pooled_signature': int(
                    duplicate_signature_components
                ),
                'maximum_distinct_component_assignment': int(matched),
                'distinct_assignment_ceiling_fraction': float(
                    matched / max(int(count), 1)
                ) if count else 1.0,
            })
            if case_index in {1, len(frame)} or case_index % 20 == 0:
                print(
                    f'  Resolution {partition} {case_index:02d}/{len(frame)}: '
                    f'{subject_id}, components={count}'
                )
    components = pd.DataFrame(component_rows)
    subjects = pd.DataFrame(subject_rows)
    summary_rows = []
    for partition in ['fold0_fit', 'fold0_heldout']:
        for tier in COMPONENT_TIERS:
            group = components[
                (components['development_partition'] == partition)
                & (components['component_tier'] == tier)
            ]
            summary_rows.append({
                'development_partition': partition,
                'component_tier': tier,
                'components': int(len(group)),
                'single_global_voxel_fraction': float(
                    group['single_global_voxel'].mean()
                ) if len(group) else 0.0,
                'global_voxel_collision_fraction': float(
                    group['shares_any_global_voxel'].mean()
                ) if len(group) else 0.0,
                'duplicate_pooled_signature_fraction': float(
                    group['duplicate_pooled_signature'].mean()
                ) if len(group) else 0.0,
                'median_global_proposal_voxels': float(
                    group['global_proposal_voxels'].median()
                ) if len(group) else 0.0,
                'mean_distinct_assignment_ceiling_fraction': float('nan'),
            })
    for partition, group in subjects.groupby('development_partition'):
        summary_rows.append({
            'development_partition': str(partition),
            'component_tier': 'subject_level_distinct_assignment_ceiling',
            'components': int(group['components'].sum()),
            'single_global_voxel_fraction': float('nan'),
            'global_voxel_collision_fraction': float('nan'),
            'duplicate_pooled_signature_fraction': float('nan'),
            'median_global_proposal_voxels': float('nan'),
            'mean_distinct_assignment_ceiling_fraction': float(
                group['distinct_assignment_ceiling_fraction'].mean()
            ),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(components, R7D_RESOLUTION_PATH)
    atomic_write_csv(summary, R7D_RESOLUTION_SUMMARY_PATH)
    return components, subjects, summary


def r7d_validation_artifacts(epoch):
    root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    paths = {
        'case': root / 'case_metrics.csv',
        'component': root / 'component_metrics.csv',
        'tier': root / 'component_tier_metrics.csv',
        'summary': root / 'validation_summary.json',
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise RuntimeError(
            'Completed validation artifacts are missing: ' + ', '.join(missing)
        )
    return paths


def r7d_paired_completed_metrics(r7_best_epoch):
    r5_paths = r7d_validation_artifacts(0)
    r7_paths = r7d_validation_artifacts(r7_best_epoch)
    r5_case = pd.read_csv(r5_paths['case'])
    r7_case = pd.read_csv(r7_paths['case'])
    case_keys = ['subject_id']
    paired_case = r5_case.merge(
        r7_case, on=case_keys, how='outer', suffixes=('_r5', '_r7'),
        validate='one_to_one', indicator=True,
    )
    if not bool((paired_case['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 case metrics do not pair one-to-one.')
    paired_case = paired_case.drop(columns=['_merge'])
    case_delta_columns = [
        'dice', 'voxel_precision', 'voxel_recall',
        'predicted_volume_ml_from_mask', 'predicted_volume_ml_from_router',
        'presence_probability', 'proposal_voxel_precision',
        'proposal_voxel_recall', 'proposal_predicted_to_target_volume_ratio',
    ]
    for name in case_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_case and right in paired_case:
            paired_case[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_case[right], errors='coerce')
                - pd.to_numeric(paired_case[left], errors='coerce')
            )

    r5_component = pd.read_csv(r5_paths['component'])
    r7_component = pd.read_csv(r7_paths['component'])
    component_keys = ['subject_id', 'component_id']
    paired_component = r5_component.merge(
        r7_component, on=component_keys, how='outer',
        suffixes=('_r5', '_r7'), validate='one_to_one', indicator=True,
    )
    if not bool((paired_component['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 component metrics do not pair one-to-one.')
    paired_component = paired_component.drop(columns=['_merge'])
    component_delta_columns = [
        'final_probability_maximum', 'proposal_probability_maximum',
        'final_maximum_component_iou', 'proposal_maximum_component_iou',
        'proposal_minimum_distance_mm',
        'budgeted_proposal_maximum_component_iou',
        'budgeted_proposal_minimum_distance_mm',
    ]
    for name in component_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_component[right], errors='coerce')
                - pd.to_numeric(paired_component[left], errors='coerce')
            )
    for name in [
        'final_detected', 'proposal_covered',
        'budgeted_proposal_within_8mm',
        'representation_failure_below_0p10',
    ]:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                paired_component[right].map(as_bool).astype(int)
                - paired_component[left].map(as_bool).astype(int)
            )
    atomic_write_csv(paired_case, R7D_PAIRED_CASE_PATH)
    atomic_write_csv(paired_component, R7D_PAIRED_COMPONENT_PATH)

    candidates = []
    positive_case_candidates = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    for label, ascending in [
        ('largest_case_dice_regression', True),
        ('largest_case_dice_improvement', False),
    ]:
        ranked = positive_case_candidates.sort_values(
            'dice_delta_r7_minus_r5', ascending=ascending
        ).head(5)
        for rank, item in enumerate(ranked.to_dict('records'), start=1):
            candidates.append({
                'candidate_reason': label,
                'rank': rank,
                'subject_id': str(item['subject_id']),
                'component_id': np.nan,
                'component_tier': '',
                'r5_value': float(item['dice_r5']),
                'r7_value': float(item['dice_r7']),
                'delta_r7_minus_r5': float(
                    item['dice_delta_r7_minus_r5']
                ),
            })
    small = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ].copy()
    small = small.sort_values(
        'proposal_probability_maximum_delta_r7_minus_r5', ascending=True
    ).head(10)
    for rank, item in enumerate(small.to_dict('records'), start=1):
        candidates.append({
            'candidate_reason': 'largest_small_proposal_probability_regression',
            'rank': rank,
            'subject_id': str(item['subject_id']),
            'component_id': int(item['component_id']),
            'component_tier': str(item['component_tier_r7']),
            'r5_value': float(item['proposal_probability_maximum_r5']),
            'r7_value': float(item['proposal_probability_maximum_r7']),
            'delta_r7_minus_r5': float(
                item['proposal_probability_maximum_delta_r7_minus_r5']
            ),
        })
    candidate_frame = pd.DataFrame(candidates)
    atomic_write_csv(candidate_frame, R7D_VISUAL_CANDIDATES_PATH)
    return paired_case, paired_component, candidate_frame


def r7d_checkpoint_pareto(r7_best_epoch):
    history = pd.read_csv(PILOT_HISTORY_PATH).sort_values('epoch').copy()
    maximize = [
        'validation_mean_dice',
        'validation_global_dice',
        'validation_small_mean_dice',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'proposal_voxel_average_precision_256bin',
    ]
    maximize = [name for name in maximize if name in history.columns]
    values = history[maximize].apply(pd.to_numeric, errors='coerce').fillna(
        -np.inf
    ).to_numpy(dtype=np.float64)
    pareto = np.ones(len(history), dtype=bool)
    for index in range(len(history)):
        dominated = np.any(
            np.all(values >= values[index], axis=1)
            & np.any(values > values[index], axis=1)
        )
        pareto[index] = not bool(dominated)
    history['pareto_nondominated'] = pareto
    history['is_selected_best_epoch'] = (
        history['epoch'].astype(int) == int(r7_best_epoch)
    )
    best_rows = {}
    metric_flags = {}
    for name in maximize + ['pilot_checkpoint_score']:
        if name not in history:
            continue
        values_series = pd.to_numeric(history[name], errors='coerce')
        best_index = values_series.idxmax()
        best_epoch = int(history.loc[best_index, 'epoch'])
        best_value = float(values_series.loc[best_index])
        flag = f'is_best_{name}'
        history[flag] = history['epoch'].astype(int) == best_epoch
        best_rows[name] = {'epoch': best_epoch, 'value': best_value}
        metric_flags[name] = flag
    atomic_write_csv(history, R7D_CHECKPOINT_PARETO_PATH)
    return history, best_rows


def r7d_branch_comparison(r5_best_epoch, r7_best_epoch):
    sources = [
        ('r5', R5_PILOT_ROOT / 'branch_reliance_history.csv', r5_best_epoch),
        ('r7', PILOT_BRANCH_HISTORY_PATH, r7_best_epoch),
    ]
    selected_frames = []
    for checkpoint, path, epoch in sources:
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        available_epochs = sorted(frame['epoch'].astype(int).unique().tolist())
        if not available_epochs:
            continue
        selected_epoch = (
            int(epoch) if int(epoch) in available_epochs
            else min(available_epochs, key=lambda value: abs(value - int(epoch)))
        )
        selected = frame[frame['epoch'].astype(int) == selected_epoch].copy()
        selected['checkpoint'] = checkpoint
        selected['requested_best_epoch'] = int(epoch)
        selected['selected_probe_epoch'] = int(selected_epoch)
        selected['exact_best_epoch_probe'] = bool(selected_epoch == int(epoch))
        selected_frames.append(selected)
    if not selected_frames:
        comparison = pd.DataFrame(columns=[
            'checkpoint', 'ablation', 'mean_target_probability_delta'
        ])
    else:
        raw = pd.concat(selected_frames, ignore_index=True)
        rows = []
        for (checkpoint, ablation), group in raw.groupby(
            ['checkpoint', 'ablation']
        ):
            rows.append({
                'checkpoint': str(checkpoint),
                'ablation': str(ablation),
                'subjects': int(group['subject_id'].nunique()),
                'requested_best_epoch': int(
                    group['requested_best_epoch'].iloc[0]
                ),
                'selected_probe_epoch': int(
                    group['selected_probe_epoch'].iloc[0]
                ),
                'exact_best_epoch_probe': bool(
                    group['exact_best_epoch_probe'].iloc[0]
                ),
                'mean_target_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_target_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_local_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_local_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_proposal_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_proposal_probability_delta'],
                    errors='coerce',
                ).mean()),
            })
        comparison = pd.DataFrame(rows)
        if set(comparison['checkpoint']) == {'r5', 'r7'}:
            pivot = comparison.pivot(
                index='ablation', columns='checkpoint',
                values='mean_target_probability_delta',
            )
            deltas = (pivot['r7'] - pivot['r5']).to_dict()
            comparison['target_delta_change_r7_minus_r5'] = comparison[
                'ablation'
            ].map(deltas)
    atomic_write_csv(comparison, R7D_BRANCH_PATH)
    return comparison


def r7d_global_component_masks(labels, count, pad_start):
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pad = np.asarray(pad_start, dtype=np.int64)
    all_coordinates = np.argwhere(labels > 0)
    full = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    if len(all_coordinates):
        pooled = np.unique((all_coordinates + pad) // downsample, axis=0)
        full[tuple(pooled.T)] = True
    components = []
    for component_id in range(1, int(count) + 1):
        coordinates = np.argwhere(labels == component_id)
        pooled = np.unique((coordinates + pad) // downsample, axis=0)
        components.append(pooled)
    return full, components


def r7d_global_domain(native_shape, pad_start):
    start = np.asarray(pad_start, dtype=np.int64)
    end = start + np.asarray(native_shape, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    low_start = start // downsample
    low_end = (end + downsample - 1) // downsample
    domain = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    slices = tuple(
        slice(int(low_start[axis]), int(low_end[axis])) for axis in range(3)
    )
    domain[slices] = True
    return domain


def r7d_component_logit_margins(
    logits, full_target, component_coordinates, domain, subject_id,
    partition, checkpoint, spacing,
):
    rows = []
    outer_radius = np.asarray(R7_OUTER_STRUCTURE.shape, dtype=np.int64) // 2
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    native_voxel_volume = voxel_volume_ml
    for component_id, pooled in enumerate(component_coordinates, start=1):
        if len(pooled) == 0:
            continue
        minimum = np.maximum(pooled.min(axis=0) - outer_radius, 0)
        maximum = np.minimum(
            pooled.max(axis=0) + outer_radius + 1,
            np.asarray(GLOBAL_INPUT_SHAPE),
        )
        slices = tuple(
            slice(int(minimum[axis]), int(maximum[axis]))
            for axis in range(3)
        )
        exact = np.zeros(tuple((maximum - minimum).tolist()), dtype=bool)
        local_coordinates = pooled - minimum
        exact[tuple(local_coordinates.T)] = True
        _, _, negative = r7_component_regions(
            exact, full_target[slices], domain[slices]
        )
        positive_logits = logits[slices][exact]
        negative_logits = logits[slices][negative]
        hard_count = min(
            len(negative_logits),
            max(R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * len(positive_logits)),
        )
        if hard_count > 0:
            hard_negative = np.partition(
                negative_logits, len(negative_logits) - hard_count
            )[-hard_count:]
            hard_negative_mean = float(hard_negative.mean())
            hard_negative_maximum = float(hard_negative.max())
        else:
            hard_negative_mean = float('nan')
            hard_negative_maximum = float('nan')
        native_voxels_estimate = int(len(pooled) * np.prod(GLOBAL_DOWNSAMPLE))
        volume_ml_estimate = native_voxels_estimate * native_voxel_volume
        positive_mean = float(positive_logits.mean())
        positive_minimum = float(positive_logits.min())
        positive_maximum = float(positive_logits.max())
        rows.append({
            'checkpoint': checkpoint,
            'development_partition': partition,
            'subject_id': subject_id,
            'component_id': int(component_id),
            'component_tier_from_pooled_extent': pilot_component_tier(
                volume_ml_estimate, native_voxels_estimate
            ),
            'global_proposal_voxels': int(len(pooled)),
            'positive_logit_mean': positive_mean,
            'positive_logit_minimum': positive_minimum,
            'positive_logit_maximum': positive_maximum,
            'hard_negative_logit_mean': hard_negative_mean,
            'hard_negative_logit_maximum': hard_negative_maximum,
            'positive_minus_hard_negative_mean_margin': (
                positive_mean - hard_negative_mean
                if np.isfinite(hard_negative_mean) else float('nan')
            ),
            'positive_minimum_minus_hard_negative_maximum_margin': (
                positive_minimum - hard_negative_maximum
                if np.isfinite(hard_negative_maximum) else float('nan')
            ),
            'positive_probability_mean': float(
                1.0 / (1.0 + np.exp(-np.clip(positive_mean, -30, 30)))
            ),
            'negative_annulus_voxels': int(negative.sum()),
            'hard_negative_voxels_used': int(hard_count),
        })
    return rows


def r7d_topk_prediction(probability, budget):
    probability = np.asarray(probability, dtype=np.float32)
    budget = int(np.clip(int(budget), 1, probability.size))
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape)


def r7d_budget_metrics(
    probability, target, spacing, predicted_volume_ml, subject_id,
    checkpoint,
):
    target = np.asarray(target, dtype=bool)
    target_voxels = int(target.sum())
    if target_voxels == 0:
        return []
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    labels, count = ndimage.label(
        target, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    component_objects = ndimage.find_objects(labels)
    component_metadata = []
    for component_id, component_slice in enumerate(component_objects, start=1):
        if component_slice is None:
            continue
        local = labels[component_slice] == component_id
        voxels = int(local.sum())
        volume_ml = voxels * voxel_volume_ml
        component_metadata.append((
            component_id, component_slice, local,
            pilot_component_tier(volume_ml, voxels),
        ))
    budgets = [
        ('router_x1', max(1, int(math.ceil(predicted_volume_ml / voxel_volume_ml)))),
        ('router_x2', max(1, int(math.ceil(2.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('router_x3', max(1, int(math.ceil(3.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('oracle_target_x3', max(1, int(math.ceil(3.0 * target_voxels)))),
    ]
    rows = []
    for budget_name, requested_budget in budgets:
        prediction = r7d_topk_prediction(probability, requested_budget)
        predicted_voxels = int(prediction.sum())
        intersection = int((prediction & target).sum())
        distance = ndimage.distance_transform_edt(
            ~prediction, sampling=spacing
        )
        within_8 = []
        small_within_8 = []
        for _, component_slice, component_local, tier in component_metadata:
            minimum_distance = float(
                distance[component_slice][component_local].min(initial=np.inf)
            )
            covered = bool(minimum_distance <= 8.0)
            within_8.append(covered)
            if tier in R7_SMALL_COMPONENT_TIERS:
                small_within_8.append(covered)
        rows.append({
            'checkpoint': checkpoint,
            'subject_id': subject_id,
            'budget_policy': budget_name,
            'router_predicted_volume_ml': float(predicted_volume_ml),
            'target_volume_ml': float(target_voxels * voxel_volume_ml),
            'router_to_target_volume_ratio': float(
                predicted_volume_ml / max(target_voxels * voxel_volume_ml, 1e-8)
            ),
            'requested_budget_voxels': int(requested_budget),
            'selected_budget_voxels': predicted_voxels,
            'selected_to_target_voxel_ratio': float(
                predicted_voxels / max(target_voxels, 1)
            ),
            'voxel_precision': float(intersection / max(predicted_voxels, 1)),
            'voxel_recall': float(intersection / max(target_voxels, 1)),
            'dice': float(
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            ),
            'components': int(len(component_metadata)),
            'component_within_8mm_fraction': float(np.mean(within_8)),
            'small_components': int(len(small_within_8)),
            'small_component_within_8mm_fraction': float(
                np.mean(small_within_8)
            ) if small_within_8 else float('nan'),
        })
    return rows


def r7d_load_model(checkpoint_path, expected_version, expected_revision, device):
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=False
    )
    checks = {
        'pilot_version': checkpoint.get('pilot_version') == expected_version,
        'code_revision': checkpoint.get('code_revision') == expected_revision,
        'parameter_count': int(checkpoint.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS,
        'ema_model_state': 'ema_model_state' in checkpoint,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'Unsafe read-only checkpoint {checkpoint_path}: '
            + ', '.join(failed)
        )
    model = VascuFateNetV22(pilot_schema).to(device)
    model.load_state_dict(checkpoint['ema_model_state'], strict=True)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    del checkpoint
    return model


@torch.no_grad()
def r7d_checkpoint_margin_and_volume_audit(
    checkpoint_label, checkpoint_path, expected_version,
    expected_revision, device,
):
    model = r7d_load_model(
        checkpoint_path, expected_version, expected_revision, device
    )
    margin_rows = []
    budget_rows = []
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            case = pilot_load_validation_case(row)
            batch = pilot_stack_sample(case['base_sample'], device)
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(model, batch)
            proposal_logits_tensor = cache['proposal_logits'].float()
            logits = proposal_logits_tensor[0, 0].cpu().numpy()
            predicted_volume_ml = float(torch.expm1(
                cache['routed']['log_volume_prediction'].float()
            )[0].clamp_min(0).cpu())
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            full_global, component_coordinates = r7d_global_component_masks(
                labels, count, case['pad_start']
            )
            domain = r7d_global_domain(case['native_shape'], case['pad_start'])
            rows = r7d_component_logit_margins(
                logits, full_global, component_coordinates, domain,
                subject_id, partition, checkpoint_label, spacing,
            )
            
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for item in rows:
                component_id = int(item['component_id'])
                voxels = int((labels == component_id).sum())
                volume_ml = voxels * voxel_volume_ml
                item['component_voxels'] = voxels
                item['component_volume_ml'] = volume_ml
                item['component_tier'] = pilot_component_tier(
                    volume_ml, voxels
                )
            margin_rows.extend(rows)

            if (
                partition == 'fold0_heldout'
                and as_bool(row['trial8_hard_metric_eligible'])
                and bool(case['target'].any())
            ):
                proposal_canvas = F.interpolate(
                    torch.sigmoid(proposal_logits_tensor),
                    size=GLOBAL_CANVAS_SHAPE,
                    mode='trilinear', align_corners=False,
                )[0, 0].cpu().numpy()
                native_slices = tuple(
                    slice(
                        case['pad_start'][axis],
                        case['pad_start'][axis] + case['native_shape'][axis],
                    )
                    for axis in range(3)
                )
                proposal_native = proposal_canvas[native_slices]
                budget_rows.extend(r7d_budget_metrics(
                    proposal_native, case['target'], case['spacing'],
                    predicted_volume_ml, subject_id, checkpoint_label,
                ))
            del batch, cache, proposal_logits_tensor, case
            if case_index in {1, len(frame)} or case_index % 10 == 0:
                print(
                    f'  {checkpoint_label} {partition} '
                    f'{case_index:02d}/{len(frame)}: {subject_id}'
                )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return margin_rows, budget_rows


def r7d_margin_summary(frame):
    rows = []
    for (checkpoint, partition, tier), group in frame.groupby([
        'checkpoint', 'development_partition', 'component_tier'
    ]):
        margin = pd.to_numeric(
            group['positive_minus_hard_negative_mean_margin'],
            errors='coerce',
        )
        rows.append({
            'checkpoint': str(checkpoint),
            'development_partition': str(partition),
            'component_tier': str(tier),
            'components': int(len(group)),
            'mean_logit_margin': float(margin.mean()),
            'median_logit_margin': float(margin.median()),
            'positive_margin_fraction': float((margin > 0).mean()),
            'mean_positive_probability': float(pd.to_numeric(
                group['positive_probability_mean'], errors='coerce'
            ).mean()),
        })
    return pd.DataFrame(rows)


def r7d_volume_budget_summary(frame):
    rows = []
    for (checkpoint, policy), group in frame.groupby(
        ['checkpoint', 'budget_policy']
    ):
        rows.append({
            'checkpoint': str(checkpoint),
            'budget_policy': str(policy),
            'subjects': int(group['subject_id'].nunique()),
            'mean_dice': float(group['dice'].mean()),
            'mean_voxel_precision': float(group['voxel_precision'].mean()),
            'mean_voxel_recall': float(group['voxel_recall'].mean()),
            'mean_component_within_8mm_fraction': float(
                group['component_within_8mm_fraction'].mean()
            ),
            'mean_small_component_within_8mm_fraction': float(
                group['small_component_within_8mm_fraction'].mean()
            ),
            'median_router_to_target_volume_ratio': float(
                group['router_to_target_volume_ratio'].median()
            ),
            'mean_selected_to_target_voxel_ratio': float(
                group['selected_to_target_voxel_ratio'].mean()
            ),
        })
    return pd.DataFrame(rows)


def r7d_small_margin_delta(summary, partition):
    subset = summary[
        (summary['development_partition'] == partition)
        & summary['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    if not len(subset):
        return float('nan')
    weighted = {}
    for checkpoint in ['r5', 'r7']:
        group = subset[subset['checkpoint'] == checkpoint]
        if not len(group):
            weighted[checkpoint] = float('nan')
            continue
        weighted[checkpoint] = float(np.average(
            group['mean_logit_margin'], weights=group['components']
        ))
    return weighted['r7'] - weighted['r5']


def run_r7_failure_mechanism_diagnostic():
    if r7d_existing_result_is_complete():
        gate = load_json(R7_FAILURE_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC ALREADY COMPLETE')
        print('=' * 80)
        print(f'Dominant mechanism: {gate["dominant_failure_mechanism"]}')
        print(f'Diagnostic gate:    {R7_FAILURE_GATE_PATH}')
        print('No source checkpoint or completed diagnostic was rewritten.')
        return

    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r7_best_epoch = int(r7_result['best_epoch'])
    r5_best_epoch = int(r5_result['best_epoch'])
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_sha_before = file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    source_paths = [
        R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH,
        R5_RESULT_GATE_PATH, PILOT_RESULT_GATE_PATH, PILOT_HISTORY_PATH,
    ]
    source_hashes_before = {
        str(path): file_sha256(path) for path in source_paths
    }
    initial_checks = {
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r7_pilot_complete': r7_result.get('pilot_complete') is True,
        'r5_checkpoint_hash_matches_result_gate': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'r7_checkpoint_hash_matches_result_gate': (
            r7_result.get('best_checkpoint_sha256') == r7_sha_before
        ),
        'r7_directional_success_was_false': (
            r7_result.get('r7_directional_success_passed') is False
        ),
        'full_fivefold_training_locked': (
            r7_result.get('full_fivefold_training_authorized') is False
        ),
        'fold0_subject_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
        ),
        'sealed_test_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
        ),
    }
    if not all(initial_checks.values()):
        failed = [name for name, passed in initial_checks.items() if not passed]
        raise RuntimeError(
            'R7 failure diagnostic upstream checks failed: '
            + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC STARTING')
    print('=' * 80)
    print(f'R5 / r7 best epochs:               {r5_best_epoch} / {r7_best_epoch}')
    print('Target access:                     fold-0 development only')
    print('Locked validation targets read:    False')
    print('Test targets read:                 False')
    print('Optimizer / backward / steps:      0 / 0 / 0')

    target_access_frame = pd.DataFrame([
        {
            'subject_id': str(row['subject_id']),
            'development_partition': partition,
            'target_arrays_authorized': True,
            'target_arrays_read_by_diagnostic': True,
            'locked_validation_target_read': False,
            'sealed_test_target_read': False,
        }
        for partition, frame in [
            ('fold0_fit', fold0_fit_df),
            ('fold0_heldout', fold0_heldout_df),
        ]
        for row in frame.to_dict('records')
    ])
    atomic_write_csv(target_access_frame, R7D_TARGET_ACCESS_PATH)

    exposure, exposure_summary = r7d_component_schedule_exposure()
    resolution, resolution_subjects, resolution_summary = (
        r7d_resolution_audit()
    )
    paired_case, paired_component, visual_candidates = (
        r7d_paired_completed_metrics(r7_best_epoch)
    )
    checkpoint_pareto, metric_best_rows = r7d_checkpoint_pareto(
        r7_best_epoch
    )
    branch_comparison = r7d_branch_comparison(
        r5_best_epoch, r7_best_epoch
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            'The train-versus-held-out r5/r7 margin audit requires CUDA.'
        )
    set_determinism(PILOT_SEED)
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    all_margin_rows = []
    all_budget_rows = []
    checkpoint_specs = [
        ('r5', R5_BEST_CHECKPOINT_PATH,
         R5_PILOT_VERSION, R5_PILOT_CODE_REVISION),
        ('r7', PILOT_BEST_CHECKPOINT_PATH,
         PILOT_VERSION, PILOT_CODE_REVISION),
    ]
    for label, path, version, revision in checkpoint_specs:
        print(f'\nRead-only global proposal audit: {label}')
        margin_rows, budget_rows = r7d_checkpoint_margin_and_volume_audit(
            label, path, version, revision, device
        )
        all_margin_rows.extend(margin_rows)
        all_budget_rows.extend(budget_rows)
    margins = pd.DataFrame(all_margin_rows)
    budgets = pd.DataFrame(all_budget_rows)
    margin_summary = r7d_margin_summary(margins)
    budget_summary = r7d_volume_budget_summary(budgets)
    atomic_write_csv(margins, R7D_MARGIN_PATH)
    atomic_write_csv(margin_summary, R7D_MARGIN_SUMMARY_PATH)
    atomic_write_csv(budgets, R7D_VOLUME_BUDGET_PATH)
    atomic_write_csv(budget_summary, R7D_VOLUME_BUDGET_SUMMARY_PATH)

    small_exposure = exposure[
        exposure['local_contrastive_eligible'].astype(bool)
    ]
    never_selected_fraction = float(
        small_exposure['never_selected'].mean()
    )
    heldout_small_resolution = resolution[
        (resolution['development_partition'] == 'fold0_heldout')
        & resolution['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    heldout_small_collision_fraction = float(
        heldout_small_resolution['shares_any_global_voxel'].mean()
    ) if len(heldout_small_resolution) else float('nan')
    heldout_assignment_ceiling = float(
        resolution_subjects[
            resolution_subjects['development_partition'] == 'fold0_heldout'
        ]['distinct_assignment_ceiling_fraction'].mean()
    )
    fit_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_fit'
    )
    heldout_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_heldout'
    )
    positive_paired_cases = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    case_dice_delta = float(
        positive_paired_cases['dice_delta_r7_minus_r5'].mean()
    )
    small_components = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ]
    small_proposal_probability_delta = float(pd.to_numeric(
        small_components[
            'proposal_probability_maximum_delta_r7_minus_r5'
        ], errors='coerce'
    ).mean())
    r7_budget = budget_summary[
        budget_summary['checkpoint'] == 'r7'
    ]
    router_x3 = r7_budget[
        r7_budget['budget_policy'] == 'router_x3'
    ]
    oracle_x3 = r7_budget[
        r7_budget['budget_policy'] == 'oracle_target_x3'
    ]
    router_small_within8 = float(
        router_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    oracle_small_within8 = float(
        oracle_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    deployment_budget_gap = oracle_small_within8 - router_small_within8
    selected_epoch = int(r7_best_epoch)
    mean_dice_best_epoch = int(
        metric_best_rows['validation_mean_dice']['epoch']
    )
    checkpoint_selection_mismatch = selected_epoch != mean_dice_best_epoch

    mechanism_evidence = {
        'component_exposure_imbalance': bool(never_selected_fraction > 0.10),
        'low_resolution_component_collision': bool(
            heldout_small_collision_fraction > 0.10
            or heldout_assignment_ceiling < 0.95
        ),
        'contrastive_train_heldout_margin_divergence': bool(
            np.isfinite(fit_small_margin_delta)
            and np.isfinite(heldout_small_margin_delta)
            and fit_small_margin_delta > heldout_small_margin_delta + 0.10
        ),
        'small_proposal_signal_regressed': bool(
            small_proposal_probability_delta < 0.0
        ),
        'deployment_volume_router_limits_coverage': bool(
            deployment_budget_gap > 0.10
        ),
        'checkpoint_objective_mismatch': bool(checkpoint_selection_mismatch),
        'r7_mean_dice_regressed': bool(case_dice_delta < 0.0),
        'r7_representation_failure_extreme': bool(
            float(r7_result['best_validation_summary'][
                'small_component_representation_failure_fraction'
            ]) > 0.75
        ),
    }
    if mechanism_evidence['contrastive_train_heldout_margin_divergence']:
        dominant_mechanism = (
            'low-resolution local-contrastive over-specialization with '
            'poor held-out satellite transfer'
        )
    elif mechanism_evidence['component_exposure_imbalance']:
        dominant_mechanism = (
            'incomplete atomic-component exposure in the stochastic curriculum'
        )
    elif mechanism_evidence['low_resolution_component_collision']:
        dominant_mechanism = (
            'global proposal-grid resolution and component-collision ceiling'
        )
    else:
        dominant_mechanism = (
            'proposal supervision/refinement mismatch rather than missing '
            'clinical or arterial routing capacity'
        )
    recommended_r8 = {
        'architecture': (
            'Preserve the r5 V2.2 global/arterial/clinical path; add a bounded '
            'native-resolution local satellite-proposal head and fuse it as '
            'a zero-initialized residual, leaving the coarse global proposal '
            'responsible for burden and broad localization.'
        ),
        'curriculum': (
            'Replace stochastic inverse-size component draws with a '
            'coverage-guaranteed component cycle before repeats, stratified '
            'by subvoxel/micro/tiny/small tier and subject.'
        ),
        'objective': (
            'Apply component contrast at native local resolution, normalize '
            'per component, cap its gradient contribution against the main '
            'segmentation loss, and retain the coarse proposal loss.'
        ),
        'deployment': (
            'Train and calibrate the predicted-volume router on fold-fit data '
            'only; use predicted rather than target volume for any inference '
            'budget and report the oracle budget only as a diagnostic ceiling.'
        ),
        'selection': (
            'Use a Pareto-safe checkpoint rule that requires no mean-Dice '
            'regression and explicit held-out small-component improvement; '
            'do not optimize the present composite score alone.'
        ),
        'next_stage': (
            'Build an R8 read-only architecture/loss preflight first, then one '
            'fold-0 pilot. Five-fold and locked validation/test evaluation '
            'remain unauthorized.'
        ),
    }

    source_hashes_after = {
        str(path): file_sha256(path) for path in source_paths
    }
    final_checks = {
        **initial_checks,
        'fit_component_count_matches_frozen_872': int(
            (resolution['development_partition'] == 'fold0_fit').sum()
        ) == 872,
        'exposure_covers_every_frozen_fit_component': len(exposure)
        == len(component_df),
        'paired_case_metrics_have_20_subjects': len(paired_case) == 20,
        'paired_component_metrics_are_nonempty': len(paired_component) > 0,
        'margin_audit_has_both_checkpoints': set(margins['checkpoint'])
        == {'r5', 'r7'},
        'margin_audit_has_fit_and_heldout': set(
            margins['development_partition']
        ) == {'fold0_fit', 'fold0_heldout'},
        'all_finite_margin_inputs': bool(np.isfinite(pd.to_numeric(
            margins['positive_logit_mean'], errors='coerce'
        )).all()),
        'deployment_budget_uses_heldout_development_only': bool(
            budgets['subject_id'].isin(fold0_heldout_df['subject_id']).all()
        ),
        'development_target_access_has_exactly_102_subjects': (
            len(target_access_frame) == 102
            and target_access_frame['subject_id'].nunique() == 102
        ),
        'development_target_access_excludes_locked_splits': bool(
            (~target_access_frame['locked_validation_target_read']).all()
            and (~target_access_frame['sealed_test_target_read']).all()
        ),
        'r5_checkpoint_unchanged': r5_sha_before
        == file_sha256(R5_BEST_CHECKPOINT_PATH),
        'r7_checkpoint_unchanged': r7_sha_before
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'all_source_artifacts_unchanged': source_hashes_before
        == source_hashes_after,
        'optimizer_not_constructed': True,
        'backward_not_called': True,
        'optimizer_steps_are_zero': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    diagnostic_passed = all(final_checks.values())

    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': R7_FAILURE_DIAGNOSTIC_VERSION,
        'code_revision': R7_FAILURE_DIAGNOSTIC_CODE_REVISION,
        'r5_best_epoch': r5_best_epoch,
        'r7_best_epoch': r7_best_epoch,
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_sha256': r7_sha_before,
        'observed_effects': {
            'mean_paired_case_dice_delta_r7_minus_r5': case_dice_delta,
            'small_proposal_probability_delta_r7_minus_r5': (
                small_proposal_probability_delta
            ),
            'eligible_components_never_selected_fraction': (
                never_selected_fraction
            ),
            'heldout_small_global_voxel_collision_fraction': (
                heldout_small_collision_fraction
            ),
            'heldout_distinct_assignment_ceiling_fraction': (
                heldout_assignment_ceiling
            ),
            'fit_small_logit_margin_delta_r7_minus_r5': (
                fit_small_margin_delta
            ),
            'heldout_small_logit_margin_delta_r7_minus_r5': (
                heldout_small_margin_delta
            ),
            'r7_router_x3_small_within_8mm_fraction': router_small_within8,
            'r7_oracle_x3_small_within_8mm_fraction': oracle_small_within8,
            'deployment_to_oracle_small_within_8mm_gap': (
                deployment_budget_gap
            ),
            'selected_checkpoint_epoch': selected_epoch,
            'mean_dice_best_epoch': mean_dice_best_epoch,
        },
        'mechanism_evidence': mechanism_evidence,
        'dominant_failure_mechanism': dominant_mechanism,
        'recommended_r8_contract': recommended_r8,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': False,
        'backward_called': False,
        'optimizer_steps': 0,
        'r8_preflight_implementation_allowed': bool(diagnostic_passed),
        'r8_training_authorized': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(summary, R7_FAILURE_SUMMARY_PATH)
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(value)}
        for name, value in final_checks.items()
    ])
    atomic_write_csv(checks_frame, R7D_CHECKS_PATH)
    output_hashes = {
        path.name: file_sha256(path) for path in R7D_OUTPUT_PATHS
    }
    gate = {
        **summary,
        'diagnostic_complete': bool(diagnostic_passed),
        'diagnostic_passed': bool(diagnostic_passed),
        'checks': final_checks,
        'source_sha256_before': source_hashes_before,
        'source_sha256_after': source_hashes_after,
        'output_sha256': output_hashes,
        'diagnostic_root': str(R7_FAILURE_DIAGNOSTIC_ROOT),
        'summary_path': str(R7_FAILURE_SUMMARY_PATH),
    }
    atomic_write_json(gate, R7_FAILURE_GATE_PATH)
    if not diagnostic_passed:
        failed = [
            name for name, passed in final_checks.items() if not passed
        ]
        raise RuntimeError(
            'TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC FAILED. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nComponent exposure by tier:')
    display(exposure_summary)
    print('\nProposal-grid resolution summary:')
    display(resolution_summary)
    print('\nTrain-versus-held-out proposal-logit margins:')
    display(margin_summary)
    print('\nDeployment-legal predicted-volume budget audit:')
    display(budget_summary)
    print('\nBranch reliance comparison:')
    display(branch_comparison)
    print('\nDiagnostic checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Dominant mechanism:                 {dominant_mechanism}')
    print(f'Eligible components never selected: {never_selected_fraction:.4f}')
    print(f'Fit small-margin change r7-r5:      {fit_small_margin_delta:+.4f}')
    print(f'Heldout small-margin change r7-r5:  {heldout_small_margin_delta:+.4f}')
    print(f'R7 router-x3 small within 8 mm:     {router_small_within8:.4f}')
    print(f'R7 oracle-x3 small within 8 mm:     {oracle_small_within8:.4f}')
    print(f'Mean-Dice best / selected epoch:    {mean_dice_best_epoch} / {selected_epoch}')
    print('Optimizer / backward / steps:       0 / 0 / 0')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('R8 preflight implementation allowed: True')
    print('R8 training authorized:             False')
    print('Full five-fold training authorized: False')
    print(f'Diagnostic summary:                 {R7_FAILURE_SUMMARY_PATH}')
    print(f'Diagnostic gate:                    {R7_FAILURE_GATE_PATH}')
    print(
        '\nNext: construct the read-only R8 dual-resolution satellite-proposal '
        'and component-cycled curriculum preflight; do not train yet.'
    )


run_r7_failure_mechanism_diagnostic()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = (
    'r7_delayed_local_contrastive_volume_bounded_r1'
)
PILOT_VERSION = 'v2_2_fold0_local_contrastive_proposal_r7'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

R7_PREFLIGHT_VERSION = 'v1_delayed_small_component_local_contrastive'
R7_PREFLIGHT_CODE_REVISION = (
    'r1_precision_bounded_diffuse_shortcut_guard'
)
R7_COMPONENT_LOSS_MAX_COEFFICIENT = 0.10
R7_COMPONENT_LOSS_START_EPOCH = 8
R7_COMPONENT_LOSS_FULL_EPOCH = 12
R7_POSITIVE_WEIGHT = 0.40
R7_HARD_NEGATIVE_WEIGHT = 0.40
R7_RANK_WEIGHT = 0.20
R7_INNER_RADIUS_MM = 4.0
R7_OUTER_RADIUS_MM = 12.0
R7_HARD_NEGATIVE_RATIO = 4
R7_HARD_NEGATIVE_MINIMUM = 16
R7_LOGIT_MARGIN = 1.0
R7_VOLUME_BUDGET_MULTIPLIER = 3.0
R7_SMALL_COMPONENT_TIERS = set(COMPONENT_TIERS[:4])
R5_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
R5_PILOT_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
R7_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip((0.8, 0.8, 2.0), GLOBAL_DOWNSAMPLE)
)

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'
R7_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r7_local_contrastive_component_proposal' / R7_PREFLIGHT_VERSION
)
R7_PREFLIGHT_GATE_PATH = (
    R7_PREFLIGHT_ROOT / 'r7_local_contrastive_preflight_gate.json'
)
R7_POLICY_PATH = R7_PREFLIGHT_ROOT / 'r7_local_contrastive_policy.json'
R5_PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / R5_PILOT_VERSION
R5_RESULT_GATE_PATH = R5_PILOT_ROOT / 'pilot_result_gate.json'
R5_BEST_CHECKPOINT_PATH = R5_PILOT_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'
R7_R5_BASELINE_DIAGNOSTIC_PATH = (
    PILOT_DIAGNOSTIC_ROOT / 'r5_volume_bounded_baseline.json'
)
R7_FAILURE_DIAGNOSTIC_VERSION = 'v1_r5_r7_failure_mechanism_audit'
R7_FAILURE_DIAGNOSTIC_CODE_REVISION = (
    'r1_exposure_resolution_margin_volume_branch_checkpoint'
)
R7_FAILURE_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'model_diagnostics'
    / MODEL_VERSION / R7_FAILURE_DIAGNOSTIC_VERSION
)
R7_FAILURE_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
R7_FAILURE_GATE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_diagnostic_gate.json'
)
R7_FAILURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_summary.json'
)



PILOT_TARGET_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'fold0_target_access_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_schedule_audit.csv'
)
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH, R7_PREFLIGHT_GATE_PATH, R7_POLICY_PATH,
    R5_RESULT_GATE_PATH, R5_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
r7_preflight_gate = load_json(R7_PREFLIGHT_GATE_PATH)
r7_policy = load_json(R7_POLICY_PATH)
r5_result_gate = load_json(R5_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r7_component_preflight_passed': (
        r7_preflight_gate.get('preflight_passed') is True
    ),
    'r7_component_preflight_all_checks_true': (
        len(r7_preflight_gate.get('checks', {})) > 0
        and all(bool(value) for value in r7_preflight_gate['checks'].values())
    ),
    'r7_preflight_version_matches': (
        r7_preflight_gate.get('preflight_version') == R7_PREFLIGHT_VERSION
    ),
    'r7_preflight_revision_matches': (
        r7_preflight_gate.get('code_revision')
        == R7_PREFLIGHT_CODE_REVISION
    ),
    'r7_implementation_allowed': (
        r7_preflight_gate.get('r7_pilot_implementation_allowed') is True
    ),
    'r7_training_was_locked_before_this_block': (
        r7_preflight_gate.get('r7_pilot_training_authorized') is False
    ),
    'r7_architecture_parameter_count_unchanged': (
        int(r7_preflight_gate.get('architecture_parameters', -1))
        == EXPECTED_V22_PARAMETERS
        and int(r7_preflight_gate.get('new_trainable_parameters', -1)) == 0
    ),
    'r7_preflight_target_scope_is_fit_only': (
        r7_preflight_gate.get('target_arrays_read_splits') == ['fold0_fit']
        and r7_preflight_gate.get('fold0_heldout_targets_read') is False
        and r7_preflight_gate.get('locked_validation_targets_read') is False
        and r7_preflight_gate.get('test_targets_read') is False
    ),
    'r7_policy_path_matches_gate': (
        str(r7_preflight_gate.get('policy_path')) == str(R7_POLICY_PATH)
    ),
    'r7_policy_local_contrastive_contract_matches': (
        r7_policy.get('selected_component_objective', {}).get(
            'eligible_tiers'
        ) == sorted(R7_SMALL_COMPONENT_TIERS)
        and int(r7_policy.get('selected_component_objective', {}).get(
            'start_epoch', -1
        )) == R7_COMPONENT_LOSS_START_EPOCH
        and int(r7_policy.get('selected_component_objective', {}).get(
            'full_coefficient_epoch', -1
        )) == R7_COMPONENT_LOSS_FULL_EPOCH
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('maximum_coefficient', float('nan'))),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('inner_radius_mm', float('nan'))),
            R7_INNER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('outer_radius_mm', float('nan'))),
            R7_OUTER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('positive_weight', float('nan'))),
            R7_POSITIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_weight', float('nan'))),
            R7_HARD_NEGATIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('rank_weight', float('nan'))),
            R7_RANK_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_ratio', -1)) == R7_HARD_NEGATIVE_RATIO
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_minimum', -1)) == R7_HARD_NEGATIVE_MINIMUM
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('logit_margin', float('nan'))),
            R7_LOGIT_MARGIN, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get('experiment_seed', -1)) == PILOT_SEED
    ),
    'r7_policy_requires_volume_bounded_diagnostics': (
        r7_policy.get('proposal_diagnostics', {}).get(
            'raw_distance_metrics_are_not_success_gates'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_proposal_precision'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_predicted_to_target_volume_ratio'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_voxel_average_precision'
        ) is True
        and math.isclose(float(r7_policy.get(
            'proposal_diagnostics', {}
        ).get('volume_budget_multiplier', float('nan'))),
            R7_VOLUME_BUDGET_MULTIPLIER, rel_tol=0.0, abs_tol=1e-12)
    ),
    'r7_policy_preserves_whole_mask_loss': (
        math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('soft_dice_weight', float('nan'))),
            0.65, rel_tol=0.0, abs_tol=1e-12,
        )
        and math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('focal_bce_weight', float('nan'))),
            0.35, rel_tol=0.0, abs_tol=1e-12,
        )
    ),
    'r7_checkpoint_policy_is_decoupled': (
        r7_policy.get('checkpoint_policy', {}).get('save_checkpoint_on')
        == 'every strict numeric score improvement'
        and r7_policy.get('checkpoint_policy', {}).get('minimum_delta_use')
        == 'early-stopping patience reset only'
        and r7_policy.get('checkpoint_policy', {}).get(
            'minimum_delta_must_not_block_checkpoint_save'
        ) is True
    ),
    'r5_read_only_baseline_gate_matches': (
        r5_result_gate.get('pilot_version') == R5_PILOT_VERSION
        and r5_result_gate.get('code_revision') == R5_PILOT_CODE_REVISION
        and r5_result_gate.get('pilot_complete') is True
        and str(r5_result_gate.get('best_checkpoint_path'))
        == str(R5_BEST_CHECKPOINT_PATH)
        and r5_result_gate.get('best_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def r7_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


R7_INNER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_INNER_RADIUS_MM
)
R7_OUTER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_OUTER_RADIUS_MM
)


def r7_component_coefficient(epoch):
    epoch = int(epoch)
    if epoch < R7_COMPONENT_LOSS_START_EPOCH:
        return 0.0
    progress = (
        (epoch - R7_COMPONENT_LOSS_START_EPOCH + 1)
        / (R7_COMPONENT_LOSS_FULL_EPOCH - R7_COMPONENT_LOSS_START_EPOCH + 1)
    )
    return float(
        R7_COMPONENT_LOSS_MAX_COEFFICIENT * np.clip(progress, 0.0, 1.0)
    )


R7_COMPONENT_COEFFICIENT_SCHEDULE = {
    epoch: r7_component_coefficient(epoch)
    for epoch in range(1, MAX_EPOCHS + 1)
}
R7_EXPECTED_RAMP = {
    8: 0.02, 9: 0.04, 10: 0.06, 11: 0.08, 12: 0.10,
}
if not all(
    R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch] == 0.0
    for epoch in range(1, R7_COMPONENT_LOSS_START_EPOCH)
):
    raise RuntimeError('R7 local-contrastive coefficient is nonzero before epoch 8.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch], expected,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch, expected in R7_EXPECTED_RAMP.items()
):
    raise RuntimeError('R7 local-contrastive coefficient ramp changed.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch],
        R7_COMPONENT_LOSS_MAX_COEFFICIENT,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch in range(R7_COMPONENT_LOSS_FULL_EPOCH, MAX_EPOCHS + 1)
):
    raise RuntimeError('R7 local-contrastive coefficient does not remain capped.')


def r7_component_regions(exact_target, full_target, valid_domain):
    exact = np.asarray(exact_target, dtype=bool)
    full = np.asarray(full_target) > 0
    domain = np.asarray(valid_domain, dtype=bool)
    inner = ndimage.binary_dilation(exact, structure=R7_INNER_STRUCTURE)
    outer = ndimage.binary_dilation(exact, structure=R7_OUTER_STRUCTURE)
    negative = outer & ~inner & domain & ~full
    return inner, outer, negative


def pilot_extract_targets(
    row, local_origin, sample_type, atomic_component_id, component_tier
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary > 0
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(binary, dtype=np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if start_a != start_b or start_a != start_c:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_domain = F.max_pool3d(
        torch.from_numpy(domain_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0

    selected_sample = str(sample_type) == 'positive'
    selected_valid = (
        selected_sample and str(component_tier) in R7_SMALL_COMPONENT_TIERS
    )
    selected_component = np.zeros_like(component_support, dtype=bool)
    if selected_sample:
        labels, count = ndimage.label(
            component_support, structure=np.ones((3, 3, 3), dtype=bool)
        )
        expected_count = len(component_records.get(subject_id, []))
        if count != expected_count:
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {expected_count}.'
            )
        component_id = int(atomic_component_id)
        if component_id < 1 or component_id > count:
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is outside '
                f'the frozen 1..{count} range.'
            )
        selected_component = labels == component_id
        if not bool(selected_component.any()):
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is empty.'
            )

    selected_canvas, selected_start = symmetric_pad(
        selected_component.astype(np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if selected_start != start_a:
        raise RuntimeError(
            f'{subject_id}: selected-component padding start disagrees.'
        )
    selected_global = F.max_pool3d(
        torch.from_numpy(selected_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0
    selected_negative = np.zeros_like(selected_global, dtype=bool)
    if selected_valid:
        _, _, selected_negative = r7_component_regions(
            selected_global, global_target[0], global_domain
        )
        if not bool(selected_negative.any()):
            raise RuntimeError(
                f'{subject_id}: selected component has no lesion-free '
                '4-12 mm negative annulus.'
            )
    if selected_sample:
        frozen = [
            item for item in component_records[subject_id]
            if int(item['atomic_component_id']) == int(atomic_component_id)
        ]
        if len(frozen) != 1:
            raise RuntimeError(
                f'{subject_id}: selected component does not map uniquely to '
                'the frozen component audit.'
            )
        expected_global_voxels = int(frozen[0]['global_proposal_voxels'])
        if int(selected_global.sum()) != expected_global_voxels:
            raise RuntimeError(
                f'{subject_id}: selected component has '
                f'{int(selected_global.sum())} pooled voxels; expected '
                f'{expected_global_voxels}.'
            )
        if bool((selected_negative & selected_global).any()):
            raise RuntimeError(
                f'{subject_id}: negative annulus overlaps exact component.'
            )
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'selected_component_target': selected_global[None].astype(bool),
        'selected_component_negative_annulus': (
            selected_negative[None].astype(bool)
        ),
        'selected_component_valid': np.asarray(
            selected_valid, dtype=bool
        ),
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'selected_component_target': targets[
                'selected_component_target'
            ],
            'selected_component_negative_annulus': targets[
                'selected_component_negative_annulus'
            ],
            'selected_component_valid': targets[
                'selected_component_valid'
            ],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def r7_local_contrastive_loss(
    logits, exact_component_target, negative_annulus_target, component_valid,
):
    exact_target = exact_component_target.bool()
    negative_target = negative_annulus_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    positive_losses = []
    negative_losses = []
    rank_losses = []
    positive_logits = []
    hard_negative_logits = []
    selected_negative_counts = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]].float()
        negative = logits[batch_index][negative_target[batch_index]].float()
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError(
                'A valid r7 component must have positive voxels and a '
                'lesion-free negative annulus.'
            )
        hard_count = min(
            int(negative.numel()),
            max(
                R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * int(positive.numel()),
            ),
        )
        hard_negative = torch.topk(
            negative, k=hard_count, largest=True
        ).values
        positive_loss = F.softplus(-positive).mean()
        negative_loss = F.softplus(hard_negative).mean()
        rank_loss = F.softplus(
            R7_LOGIT_MARGIN - positive.mean() + hard_negative.mean()
        )
        contrastive_loss = (
            R7_POSITIVE_WEIGHT * positive_loss
            + R7_HARD_NEGATIVE_WEIGHT * negative_loss
            + R7_RANK_WEIGHT * rank_loss
        )
        losses.append(contrastive_loss)
        positive_losses.append(positive_loss)
        negative_losses.append(negative_loss)
        rank_losses.append(rank_loss)
        positive_logits.append(positive.mean())
        hard_negative_logits.append(hard_negative.mean())
        selected_negative_counts.append(hard_count)

    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'local_contrastive_loss': zero,
            'component_positive_loss': zero,
            'component_hard_negative_loss': zero,
            'component_rank_loss': zero,
            'mean_component_positive_logit': zero.detach(),
            'mean_component_hard_negative_logit': zero.detach(),
            'selected_hard_negative_voxels': zero.detach(),
            'selected_components_in_batch': zero.detach(),
        }
    return torch.stack(losses).mean(), {
        'local_contrastive_loss': torch.stack(losses).mean(),
        'component_positive_loss': torch.stack(positive_losses).mean(),
        'component_hard_negative_loss': torch.stack(negative_losses).mean(),
        'component_rank_loss': torch.stack(rank_losses).mean(),
        'mean_component_positive_logit': torch.stack(positive_logits).mean(),
        'mean_component_hard_negative_logit': torch.stack(
            hard_negative_logits
        ).mean(),
        'selected_hard_negative_voxels': torch.as_tensor(
            float(sum(selected_negative_counts)), device=logits.device
        ),
        'selected_components_in_batch': torch.as_tensor(
            float(len(losses)), device=logits.device
        ),
    }


def r7_local_contrastive_proposal_loss(
    logits, full_target, exact_component_target,
    negative_annulus_target, component_valid, epoch,
):
    whole_dice = pilot_soft_dice_loss(logits, full_target).mean()
    whole_bce = pilot_focal_bce_loss(logits, full_target).mean()
    whole_loss = 0.65 * whole_dice + 0.35 * whole_bce
    contrastive_loss, contrastive_parts = r7_local_contrastive_loss(
        logits, exact_component_target, negative_annulus_target,
        component_valid,
    )
    coefficient = r7_component_coefficient(epoch)
    proposal_loss = whole_loss + coefficient * contrastive_loss
    return proposal_loss, {
        'proposal_whole_mask_loss': whole_loss,
        'proposal_dice_loss': whole_dice,
        'proposal_focal_bce_loss': whole_bce,
        **contrastive_parts,
        'component_loss_coefficient': torch.as_tensor(
            coefficient, device=logits.device, dtype=torch.float32
        ),
    }


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(
    model, cpu_batch, device, epoch, batch_index,
    dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    if dropout_flags_override is not None:
        required_flags = {
            'force_atlas_dropout', 'force_vascular_dropout',
            'force_occlusion_dropout', 'force_clinical_dropout',
            'force_global_dropout', 'force_arterial_dropout',
        }
        if set(dropout_flags_override) != required_flags:
            raise RuntimeError(
                'Dropout override keys do not match the frozen branch set.'
            )
        flags = {
            name: bool(dropout_flags_override[name])
            for name in sorted(required_flags)
        }
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_loss, proposal_parts = r7_local_contrastive_proposal_loss(
        output['global_proposal_logits'], global_target,
        batch['selected_component_target'],
        batch['selected_component_negative_annulus'],
        batch['selected_component_valid'], epoch,
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        **proposal_parts,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'arterial_valid_nodes': arterial_valid.sum().detach().float(),
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'arterial_dropped': torch.as_tensor(float(flags['force_arterial_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_volume_bounded_prediction(probability, target_voxels):
    probability = np.asarray(probability, dtype=np.float32)
    budget = min(
        probability.size,
        max(1, int(math.ceil(
            R7_VOLUME_BUDGET_MULTIPLIER * int(target_voxels)
        ))),
    )
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape), budget


def pilot_histogram_average_precision(positive_histogram, negative_histogram):
    positive = np.asarray(positive_histogram, dtype=np.float64)[::-1]
    negative = np.asarray(negative_histogram, dtype=np.float64)[::-1]
    total_positive = float(positive.sum())
    if total_positive <= 0:
        return float('nan')
    cumulative_positive = np.cumsum(positive)
    cumulative_total = np.cumsum(positive + negative)
    precision = cumulative_positive / np.maximum(cumulative_total, 1.0)
    return float((precision * positive).sum() / total_positive)


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    voxel_precision = total_intersection / max(total_prediction, 1)
    predicted_to_target_volume_ratio = (
        total_prediction / max(total_target, 1)
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'voxel_precision': float(voxel_precision),
        'predicted_to_target_volume_ratio': float(
            predicted_to_target_volume_ratio
        ),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['volume_calibration_score'] = np.exp(-np.abs(
        np.log(np.maximum(
            proposal_threshold_df['predicted_to_target_volume_ratio'], 1e-6
        ))
    ))
    proposal_threshold_df['proposal_selection_score'] = (
        0.35 * proposal_threshold_df['positive_global_dice']
        + 0.15 * proposal_threshold_df['lesion_f1']
        + 0.20 * proposal_threshold_df['voxel_precision']
        + 0.15 * proposal_threshold_df['component_recall']
        + 0.15 * proposal_threshold_df['volume_calibration_score']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        [
            'proposal_selection_score', 'positive_global_dice',
            'voxel_precision', 'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    proposal_ap_positive_histogram = np.zeros(256, dtype=np.int64)
    proposal_ap_negative_histogram = np.zeros(256, dtype=np.int64)
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        proposal_intersection = int((proposal_prediction & target).sum())
        proposal_predicted_voxels = int(proposal_prediction.sum())
        proposal_distance = None
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        budgeted_proposal_prediction = None
        budgeted_proposal_distance = None
        proposal_volume_budget_voxels = 0
        if hard_eligible and target_voxels > 0:
            (
                budgeted_proposal_prediction,
                proposal_volume_budget_voxels,
            ) = pilot_volume_bounded_prediction(proposal, target_voxels)
            budgeted_proposal_distance = ndimage.distance_transform_edt(
                ~budgeted_proposal_prediction, sampling=case['spacing']
            )
            proposal_ap_positive_histogram += np.histogram(
                proposal[target], bins=256, range=(0.0, 1.0)
            )[0]
            proposal_ap_negative_histogram += np.histogram(
                proposal[~target], bins=256, range=(0.0, 1.0)
            )[0]
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
            'proposal_predicted_voxels': proposal_predicted_voxels,
            'proposal_intersection_voxels': proposal_intersection,
            'proposal_voxel_precision': (
                proposal_intersection / max(proposal_predicted_voxels, 1)
            ),
            'proposal_voxel_recall': (
                proposal_intersection / max(target_voxels, 1)
            ),
            'proposal_predicted_to_target_volume_ratio': (
                proposal_predicted_voxels / max(target_voxels, 1)
            ),
            'proposal_volume_budget_voxels': proposal_volume_budget_voxels,
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            budgeted_proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    budgeted_proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                budgeted_proposal_max_iou = float(
                    budgeted_proposal_iou[component_id][
                        'maximum_component_iou'
                    ]
                )
                proposal_distance_mm = float('nan')
                if proposal_distance is not None:
                    proposal_distance_mm = float(
                        proposal_distance[component_slice][component_local].min(
                            initial=np.inf
                        )
                    )
                budgeted_proposal_distance_mm = float(
                    budgeted_proposal_distance[component_slice][
                        component_local
                    ].min(initial=np.inf)
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                budgeted_proposal_covered = (
                    budgeted_proposal_max_iou >= 0.05
                )
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += int(budgeted_proposal_covered)
                tier_counts[tier][3] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'proposal_minimum_distance_mm': proposal_distance_mm,
                    'budgeted_proposal_maximum_component_iou': (
                        budgeted_proposal_max_iou
                    ),
                    'budgeted_proposal_minimum_distance_mm': (
                        budgeted_proposal_distance_mm
                    ),
                    'proposal_within_2mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 2.0
                    ),
                    'proposal_within_4mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 4.0
                    ),
                    'proposal_within_8mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_within_2mm': bool(
                        budgeted_proposal_distance_mm <= 2.0
                    ),
                    'budgeted_proposal_within_4mm': bool(
                        budgeted_proposal_distance_mm <= 4.0
                    ),
                    'budgeted_proposal_within_8mm': bool(
                        budgeted_proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_covered': bool(
                        budgeted_proposal_covered
                    ),
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, budgeted_proposal_tp, total = tier_counts[tier]
        tier_components = [
            item for item in component_rows
            if item['component_tier'] == tier
        ]
        finite_distances = [
            float(item['proposal_minimum_distance_mm'])
            for item in tier_components
            if np.isfinite(item['proposal_minimum_distance_mm'])
        ]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
            'budgeted_proposal_component_recall': (
                budgeted_proposal_tp / max(total, 1)
            ),
            'proposal_within_2mm_fraction': float(np.mean([
                item['proposal_within_2mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_4mm_fraction': float(np.mean([
                item['proposal_within_4mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_8mm_fraction': float(np.mean([
                item['proposal_within_8mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_2mm_fraction': float(np.mean([
                item['budgeted_proposal_within_2mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_4mm_fraction': float(np.mean([
                item['budgeted_proposal_within_4mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_8mm_fraction': float(np.mean([
                item['budgeted_proposal_within_8mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'median_finite_proposal_distance_mm': float(
                np.median(finite_distances)
            ) if finite_distances else float('nan'),
            'mean_proposal_maximum_probability': float(np.mean([
                item['proposal_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'mean_final_maximum_probability': float(np.mean([
                item['final_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'representation_failure_fraction': float(np.mean([
                item['representation_failure_below_0p10']
                for item in tier_components
            ])) if tier_components else 0.0,
        })
    small_component_rows = [
        item for item in component_rows
        if item['component_tier'] in set(COMPONENT_TIERS[:4])
    ]
    proposal_voxel_average_precision = pilot_histogram_average_precision(
        proposal_ap_positive_histogram, proposal_ap_negative_histogram
    )
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'proposal_voxel_precision': float(
            proposal_selected['voxel_precision']
        ),
        'proposal_predicted_to_target_volume_ratio': float(
            proposal_selected['predicted_to_target_volume_ratio']
        ),
        'proposal_volume_calibration_score': float(
            proposal_selected['volume_calibration_score']
        ),
        'proposal_voxel_average_precision_256bin': float(
            proposal_voxel_average_precision
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'small_components_audited': len(small_component_rows),
        'small_component_proposal_within_2mm_fraction': float(np.mean([
            item['proposal_within_2mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_4mm_fraction': float(np.mean([
            item['proposal_within_4mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_8mm_fraction': float(np.mean([
            item['proposal_within_8mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_budgeted_proposal_within_2mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_2mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_4mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_4mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_8mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_8mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered'] for item in component_rows
        ])) if component_rows else 0.0,
        'small_budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_representation_failure_fraction': float(np.mean([
            item['representation_failure_below_0p10']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    required_finite_summary_metrics = [
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'proposal_global_dice', 'proposal_component_recall',
        'proposal_voxel_precision',
        'proposal_predicted_to_target_volume_ratio',
        'proposal_voxel_average_precision_256bin',
        'budgeted_proposal_component_recall',
        'small_budgeted_proposal_component_recall',
        'small_component_budgeted_proposal_within_2mm_fraction',
        'small_component_budgeted_proposal_within_4mm_fraction',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'small_component_representation_failure_fraction',
        'pilot_checkpoint_score',
    ]
    nonfinite_summary_metrics = [
        name for name in required_finite_summary_metrics
        if not np.isfinite(float(summary[name]))
    ]
    if nonfinite_summary_metrics:
        raise RuntimeError(
            'Validation produced non-finite required metrics: '
            + ', '.join(nonfinite_summary_metrics)
        )
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
            and completed.get('best_checkpoint_sha256')
            == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R7 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_index = None
    smoke_subject_id = None
    for candidate_index, entry in enumerate(dataset.schedule):
        if entry['sample_type'] != 'positive':
            continue
        candidate_subject = str(entry['subject_id'])
        if not as_bool(
            fit_row_by_subject[candidate_subject]['trial8_atlas_enabled']
        ):
            continue
        candidate_sample = dataset[candidate_index]
        candidate_checks = {
            'atlas_enabled': bool(np.asarray(
                candidate_sample['atlas_enabled']
            ).reshape(-1)[0]),
            'territory_nodes_valid': bool(np.asarray(
                candidate_sample['territory_valid']
            ).astype(bool).any()),
            'vessel_nodes_valid': bool(np.asarray(
                candidate_sample['vessel_valid']
            ).astype(bool).any()),
            'selected_component_valid': bool(np.asarray(
                candidate_sample['selected_component_valid']
            ).reshape(-1)[0]),
        }
        if all(candidate_checks.values()):
            smoke_index = int(candidate_index)
            smoke_subject_id = candidate_subject
            break
    if smoke_index is None:
        raise RuntimeError(
            'No positive fold-0-fit sample has simultaneous atlas, territory, '
            'vessel, and selected-component support for the all-branch smoke '
            'test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    smoke_dropout_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device,
            epoch=R7_COMPONENT_LOSS_FULL_EPOCH, batch_index=0,
            dropout_flags_override=smoke_dropout_flags,
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    r7_smoke_checks = {
        'positive_sample_selected': bool(
            smoke_batch['selected_component_valid'].bool().all().item()
        ),
        'atlas_enabled_for_smoke': bool(
            smoke_batch['atlas_enabled'].bool().all().item()
        ),
        'territory_support_for_smoke': bool(
            smoke_batch['territory_valid'].bool().any().item()
        ),
        'vessel_support_for_smoke': bool(
            smoke_batch['vessel_valid'].bool().any().item()
        ),
        'projected_arterial_nodes_valid_for_smoke': float(
            smoke_parts['arterial_valid_nodes'].detach().float().cpu()
        ) > 0.0,
        'arterial_loss_finite_for_smoke': bool(torch.isfinite(
            smoke_parts['arterial_loss']
        ).item()),
        'all_branch_dropouts_disabled_for_smoke': not any(
            smoke_dropout_flags.values()
        ),
        'exact_component_target_nonempty': bool(
            smoke_batch['selected_component_target'].bool().any().item()
        ),
        'negative_annulus_nonempty': bool(
            smoke_batch['selected_component_negative_annulus']
            .bool().any().item()
        ),
        'negative_annulus_disjoint_from_exact_component': not bool(torch.any(
            smoke_batch['selected_component_target'].bool()
            & smoke_batch['selected_component_negative_annulus'].bool()
        ).item()),
        'local_contrastive_loss_positive': float(
            smoke_parts['local_contrastive_loss']
            .detach().float().cpu()
        ) > 0.0,
        'smoke_uses_maximum_contrastive_coefficient': math.isclose(
            float(smoke_parts['component_loss_coefficient']
                  .detach().float().cpu()),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT,
            rel_tol=0.0, abs_tol=1e-7,
        ),
        'whole_mask_proposal_loss_finite': bool(torch.isfinite(
            smoke_parts['proposal_whole_mask_loss']
        ).item()),
    }
    if not all(r7_smoke_checks.values()):
        failed = [
            name for name, passed in r7_smoke_checks.items() if not passed
        ]
        raise RuntimeError(
            'R7 local-contrastive smoke checks failed: ' + ', '.join(failed)
        )
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    if R7_R5_BASELINE_DIAGNOSTIC_PATH.exists():
        r5_baseline_diagnostic = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
        if (
            r5_baseline_diagnostic.get('r5_checkpoint_sha256')
            != r5_checkpoint_sha256
            or r5_baseline_diagnostic.get('metric_contract_revision')
            != PILOT_CODE_REVISION
        ):
            raise RuntimeError(
                'Existing r5 volume-bounded baseline is incompatible with '
                'the current r7 contract.'
            )
        r5_recomputed_summary = dict(
            r5_baseline_diagnostic['validation_summary']
        )
    else:
        rng_state_before_baseline = pilot_capture_rng_state()
        r5_checkpoint = torch.load(
            R5_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
        )
        r5_checkpoint_checks = {
            'pilot_version': (
                r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
            ),
            'code_revision': (
                r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
            ),
            'parameter_count': int(
                r5_checkpoint.get('parameter_count', -1)
            ) == EXPECTED_V22_PARAMETERS,
        }
        if not all(r5_checkpoint_checks.values()):
            failed = [
                name for name, passed in r5_checkpoint_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Unsafe r5 read-only baseline checkpoint: '
                + ', '.join(failed)
            )
        r5_baseline_model = VascuFateNetV22(pilot_schema).to(device).eval()
        r5_baseline_model.load_state_dict(r5_checkpoint['ema_model_state'])
        for parameter in r5_baseline_model.parameters():
            parameter.requires_grad_(False)
        print(
            '\nRecomputing the completed r5 best checkpoint with the r7 '
            'precision- and volume-bounded diagnostic contract.'
        )
        r5_recomputed_summary, r5_recomputed_tiers = pilot_validate(
            r5_baseline_model, epoch=0, device=device
        )
        r5_baseline_diagnostic = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'metric_contract_revision': PILOT_CODE_REVISION,
            'r5_pilot_version': R5_PILOT_VERSION,
            'r5_code_revision': R5_PILOT_CODE_REVISION,
            'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'validation_summary': r5_recomputed_summary,
            'component_tier_metrics': r5_recomputed_tiers,
            'r5_checkpoint_loaded_into_r7_model': False,
            'optimizer_steps': 0,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        }
        atomic_write_json(
            r5_baseline_diagnostic, R7_R5_BASELINE_DIAGNOSTIC_PATH
        )
        del r5_baseline_model, r5_checkpoint
        gc.collect()
        torch.cuda.empty_cache()
        pilot_restore_rng_state(rng_state_before_baseline)

    
    
    
    
    del model, ema_model
    gc.collect()
    torch.cuda.empty_cache()
    set_determinism(PILOT_SEED)
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed during lineage reset.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    lineage_smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True
    )
    lineage_smoke_batch = next(iter(lineage_smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        lineage_smoke_loss, lineage_smoke_parts = (
            pilot_compute_training_loss(
                model, lineage_smoke_batch, device, epoch=1, batch_index=0
            )
        )
    if not bool(torch.isfinite(lineage_smoke_loss).item()):
        raise RuntimeError('R7 controlled-lineage smoke loss is non-finite.')
    if float(
        lineage_smoke_parts['component_loss_coefficient']
        .detach().float().cpu()
    ) != 0.0:
        raise RuntimeError(
            'R7 controlled-lineage epoch-1 coefficient is not exactly zero.'
        )
    lineage_smoke_loss.backward()
    lineage_gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            parameter.grad
            for name, parameter in model.named_parameters()
            if name.startswith(prefix) and parameter.grad is not None
        ]
        lineage_gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(gradient).all().item())
            for gradient in gradients
        ) and any(
            float(gradient.detach().float().norm().cpu()) > 0.0
            for gradient in gradients
        )
    model.zero_grad(set_to_none=True)
    if not all(lineage_gradient_checks.values()):
        failed = [
            name for name, passed in lineage_gradient_checks.items()
            if not passed
        ]
        raise RuntimeError(
            'R7 controlled-lineage gradient smoke failed: '
            + ', '.join(failed)
        )

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r7_last_checkpoint_only',
        'strict_best_checkpoint_policy': (
            'save every strict numeric pilot-score improvement'
        ),
        'minimum_delta_policy': (
            'used only to reset early-stopping patience'
        ),
        'r7_preflight_gate_sha256': file_sha256(R7_PREFLIGHT_GATE_PATH),
        'r7_proposal_loss_policy_sha256': file_sha256(R7_POLICY_PATH),
        'component_loss_start_epoch': R7_COMPONENT_LOSS_START_EPOCH,
        'component_loss_full_epoch': R7_COMPONENT_LOSS_FULL_EPOCH,
        'component_loss_maximum_coefficient': (
            R7_COMPONENT_LOSS_MAX_COEFFICIENT
        ),
        'component_positive_weight': R7_POSITIVE_WEIGHT,
        'component_hard_negative_weight': R7_HARD_NEGATIVE_WEIGHT,
        'component_rank_weight': R7_RANK_WEIGHT,
        'component_inner_radius_mm': R7_INNER_RADIUS_MM,
        'component_outer_radius_mm': R7_OUTER_RADIUS_MM,
        'component_logit_margin': R7_LOGIT_MARGIN,
        'component_eligible_tiers': sorted(R7_SMALL_COMPONENT_TIERS),
        'component_coefficient_schedule': {
            str(epoch): coefficient
            for epoch, coefficient in R7_COMPONENT_COEFFICIENT_SCHEDULE.items()
        },
        'proposal_volume_budget_multiplier': R7_VOLUME_BUDGET_MULTIPLIER,
        'proposal_volume_budget_policy': (
            'diagnostic only: retain the top 3x ground-truth target voxels; '
            'never used as an inference rule or checkpoint-selection input'
        ),
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'controlled_lineage_matches_r5_seed': True,
        'controlled_lineage_epoch1_component_coefficient': 0.0,
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'controlled_lineage_gradient_checks': lineage_gradient_checks,
        'controlled_lineage_smoke_loss': float(
            lineage_smoke_loss.detach().cpu()
        ),
        'r7_selected_component_smoke_checks': r7_smoke_checks,
        'r7_component_coefficient_schedule_verified': True,
        'r7_smoke_subject_id': smoke_subject_id,
        'r7_smoke_schedule_index': smoke_index,
        'r7_smoke_dropout_flags': smoke_dropout_flags,
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R7 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R7 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r7 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Controlled r5-seed lineage:          verified')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('Local contrastive rescue:              small tiers, epochs 8-40')
    print('Maximum contrastive coefficient:       0.10')
    print('Proposal anti-shortcut diagnostics:    oracle-volume-bounded (audit only)')
    print('R5 recomputed mean/global Dice:        '
          f'{r5_recomputed_summary["positive_mean_dice"]:.4f} / '
          f'{r5_recomputed_summary["positive_global_dice"]:.4f}')
    print('R5 oracle-budgeted small within 8 mm:  '
          f'{r5_recomputed_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}')
    print('Best-checkpoint minimum delta:        disabled (strict numeric best)')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        strict_best_improved = current_score > best_score
        patience_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if strict_best_improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
        if patience_improved:
            patience_reference_score = current_score
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        epoch_tier_metrics = {
            str(row['component_tier']): row for row in tier_rows
        }
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'proposal_voxel_precision': summary['proposal_voxel_precision'],
            'proposal_predicted_to_target_volume_ratio': summary[
                'proposal_predicted_to_target_volume_ratio'
            ],
            'proposal_voxel_average_precision_256bin': summary[
                'proposal_voxel_average_precision_256bin'
            ],
            'budgeted_proposal_component_recall': summary[
                'budgeted_proposal_component_recall'
            ],
            'small_component_proposal_within_2mm_fraction': summary[
                'small_component_proposal_within_2mm_fraction'
            ],
            'small_component_proposal_within_4mm_fraction': summary[
                'small_component_proposal_within_4mm_fraction'
            ],
            'small_component_proposal_within_8mm_fraction': summary[
                'small_component_proposal_within_8mm_fraction'
            ],
            'small_component_budgeted_proposal_within_8mm_fraction': summary[
                'small_component_budgeted_proposal_within_8mm_fraction'
            ],
            'small_component_representation_failure_fraction': summary[
                'small_component_representation_failure_fraction'
            ],
            **{
                f'{tier_name}_{metric_name}': float(
                    epoch_tier_metrics.get(tier_name, {}).get(
                        metric_name, float('nan')
                    )
                )
                for tier_name in COMPONENT_TIERS
                for metric_name in (
                    'final_component_recall',
                    'proposal_component_recall',
                    'budgeted_proposal_component_recall',
                    'proposal_within_8mm_fraction',
                    'budgeted_proposal_within_8mm_fraction',
                    'representation_failure_fraction',
                )
            },
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'strict_best_improved': strict_best_improved,
            'patience_improved_by_minimum_delta': patience_improved,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if strict_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'contrastive={running["local_contrastive_loss"]/max(processed,1):.4f}, '
            f'coef={running["component_loss_coefficient"]/max(processed,1):.3f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        print(
            '  R7 raw proposal geometry: small within 2/4/8 mm='
            f'{summary["small_component_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_8mm_fraction"]:.4f}, '
            'small representation failure='
            f'{summary["small_component_representation_failure_fraction"]:.4f}'
        )
        print(
            '  Oracle-volume-bounded diagnostic: small within 2/4/8 mm='
            f'{summary["small_component_budgeted_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}, '
            f'component recall={summary["budgeted_proposal_component_recall"]:.4f}, '
            f'precision/AP={summary["proposal_voxel_precision"]:.4f}/'
            f'{summary["proposal_voxel_average_precision_256bin"]:.4f}, '
            f'volume ratio={summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_baseline = r7_policy['r5_baseline']
    success_policy = r7_policy['r7_directional_success_gates']
    best_tier_by_name = {
        str(row['component_tier']): row for row in best_tier_rows
    }
    best_large_proposal_recall = float(
        best_tier_by_name.get('large', {}).get(
            'proposal_component_recall', 0.0
        )
    )
    mean_dice_improvement = (
        float(best_summary['positive_mean_dice'])
        - float(r5_recomputed_summary['positive_mean_dice'])
    )
    budgeted_small_within_8mm_improvement = (
        float(best_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
        - float(r5_recomputed_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
    )
    small_representation_failure_reduction = (
        float(r5_recomputed_summary[
            'small_component_representation_failure_fraction'
        ])
        - float(best_summary[
            'small_component_representation_failure_fraction'
        ])
    )
    global_dice_regression = (
        float(r5_recomputed_summary['positive_global_dice'])
        - float(best_summary['positive_global_dice'])
    )
    proposal_dice_regression = (
        float(r5_recomputed_summary['proposal_global_dice'])
        - float(best_summary['proposal_global_dice'])
    )
    r7_success_checks = {
        'mean_dice_does_not_materially_regress': (
            -mean_dice_improvement
            <= float(success_policy[
                'mean_dice_maximum_allowed_regression'
            ])
        ),
        'mean_dice_reaches_target_improvement': (
            mean_dice_improvement
            >= float(success_policy[
                'mean_dice_target_absolute_improvement'
            ])
        ),
        'volume_bounded_small_within_8mm_improves': (
            budgeted_small_within_8mm_improvement
            >= float(success_policy[
                'volume_bounded_small_within_8mm_absolute_improvement_at_least'
            ])
        ),
        'small_representation_failure_reduced_by_required_margin': (
            small_representation_failure_reduction
            >= float(success_policy[
                'small_representation_failure_absolute_reduction_at_least'
            ])
        ),
        'proposal_component_recall_meets_minimum': (
            float(best_summary['proposal_component_recall'])
            >= float(success_policy['proposal_component_recall_minimum'])
        ),
        'global_dice_regression_within_limit': (
            global_dice_regression
            <= float(success_policy[
                'global_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_dice_regression_within_limit': (
            proposal_dice_regression
            <= float(success_policy[
                'proposal_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_volume_ratio_is_bounded': (
            float(best_summary[
                'proposal_predicted_to_target_volume_ratio'
            ])
            <= float(success_policy[
                'proposal_predicted_to_target_volume_ratio_maximum'
            ])
        ),
        'empty_false_positive_volume_within_limit': (
            float(best_summary['empty_false_positive_volume_ml'])
            <= float(success_policy[
                'mean_false_positive_volume_ml_maximum'
            ])
        ),
        'large_component_proposal_recall_meets_minimum': (
            best_large_proposal_recall
            >= float(success_policy[
                'large_component_proposal_recall_minimum'
            ])
        ),
    }
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'r5_baseline_evidence': r5_baseline,
        'r5_recomputed_under_r7_metric_contract': r5_recomputed_summary,
        'r5_read_only_baseline_diagnostic_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r7_directional_success_policy': success_policy,
        'r7_directional_success_checks': r7_success_checks,
        'r7_directional_success_passed': all(r7_success_checks.values()),
        'r7_directional_effects': {
            'mean_dice_absolute_change': mean_dice_improvement,
            'volume_bounded_small_within_8mm_absolute_change': (
                budgeted_small_within_8mm_improvement
            ),
            'small_representation_failure_absolute_reduction': (
                small_representation_failure_reduction
            ),
            'global_dice_regression': global_dice_regression,
            'proposal_dice_regression': proposal_dice_regression,
            'proposal_predicted_to_target_volume_ratio': float(
                best_summary['proposal_predicted_to_target_volume_ratio']
            ),
            'proposal_voxel_average_precision_256bin': float(
                best_summary['proposal_voxel_average_precision_256bin']
            ),
            'large_component_proposal_recall': (
                best_large_proposal_recall
            ),
        },
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(
        'Raw small proposal within 8 mm:   '
        f'{best_summary["small_component_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Oracle-budgeted small within 8 mm: '
        f'{best_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Proposal precision / AP:          '
        f'{best_summary["proposal_voxel_precision"]:.4f} / '
        f'{best_summary["proposal_voxel_average_precision_256bin"]:.4f}'
    )
    print(
        'Proposal predicted/target ratio:  '
        f'{best_summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
    )
    print(
        'Mean Dice change vs recomputed r5:'
        f' {mean_dice_improvement:+.4f}'
    )
    print(
        'Budgeted within-8-mm change vs r5:'
        f' {budgeted_small_within_8mm_improvement:+.4f}'
    )
    print(
        'Small representation failure:     '
        f'{best_summary["small_component_representation_failure_fraction"]:.4f}'
    )
    print(
        'R7 directional success gates:     '
        f'{all(r7_success_checks.values())}'
    )
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')






R7D_EXPOSURE_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure.csv'
R7D_EXPOSURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure_by_tier.csv'
)
R7D_RESOLUTION_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_component_audit.csv'
)
R7D_RESOLUTION_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_summary.csv'
)
R7D_PAIRED_CASE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_case_metrics.csv'
)
R7D_PAIRED_COMPONENT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_component_metrics.csv'
)
R7D_MARGIN_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_train_heldout_margin_audit.csv'
)
R7D_MARGIN_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_margin_summary.csv'
)
R7D_VOLUME_BUDGET_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_audit.csv'
)
R7D_VOLUME_BUDGET_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_summary.csv'
)
R7D_CHECKPOINT_PARETO_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'checkpoint_metric_pareto.csv'
)
R7D_BRANCH_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'branch_reliance_comparison.csv'
)
R7D_VISUAL_CANDIDATES_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'visual_qa_candidates.csv'
)
R7D_CHECKS_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'diagnostic_checks.csv'
R7D_TARGET_ACCESS_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'development_target_access_audit.csv'
)

R7D_OUTPUT_PATHS = [
    R7D_EXPOSURE_PATH,
    R7D_EXPOSURE_SUMMARY_PATH,
    R7D_RESOLUTION_PATH,
    R7D_RESOLUTION_SUMMARY_PATH,
    R7D_PAIRED_CASE_PATH,
    R7D_PAIRED_COMPONENT_PATH,
    R7D_MARGIN_PATH,
    R7D_MARGIN_SUMMARY_PATH,
    R7D_VOLUME_BUDGET_PATH,
    R7D_VOLUME_BUDGET_SUMMARY_PATH,
    R7D_CHECKPOINT_PARETO_PATH,
    R7D_BRANCH_PATH,
    R7D_VISUAL_CANDIDATES_PATH,
    R7D_CHECKS_PATH,
    R7D_TARGET_ACCESS_PATH,
    R7_FAILURE_SUMMARY_PATH,
]


def r7d_existing_result_is_complete():
    if not R7_FAILURE_GATE_PATH.exists():
        return False
    gate = load_json(R7_FAILURE_GATE_PATH)
    if not (
        gate.get('diagnostic_complete') is True
        and gate.get('diagnostic_version') == R7_FAILURE_DIAGNOSTIC_VERSION
        and gate.get('code_revision') == R7_FAILURE_DIAGNOSTIC_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_checkpoint_sha256')
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    ):
        return False
    output_hashes = gate.get('output_sha256', {})
    return all(
        path.exists()
        and output_hashes.get(path.name) == file_sha256(path)
        for path in R7D_OUTPUT_PATHS
    )


def r7d_gini(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0 or float(values.sum()) <= 0.0:
        return 0.0
    values = np.sort(values)
    index = np.arange(1, values.size + 1, dtype=np.float64)
    return float(
        np.sum((2.0 * index - values.size - 1.0) * values)
        / (values.size * values.sum())
    )


def r7d_component_support(row):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    mode = str(row['trial8_supervision_mode'])
    if mode == 'hard_and_soft':
        support = binary
    elif mode == 'soft_only_subvoxel':
        support = soft > 0
    elif mode == 'true_empty':
        support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unsupported supervision mode {mode!r}.'
        )
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: non-finite target arrays.')
    return binary, soft, support, spacing


def r7d_component_schedule_exposure():
    counts = defaultdict(int)
    first_epoch = {}
    last_epoch = {}
    for epoch in range(R7_COMPONENT_LOSS_START_EPOCH, MAX_EPOCHS + 1):
        schedule = pilot_build_epoch_schedule(epoch)
        for entry in schedule:
            if (
                str(entry['sample_type']) != 'positive'
                or str(entry['component_tier']) not in R7_SMALL_COMPONENT_TIERS
            ):
                continue
            key = (
                str(entry['subject_id']), int(entry['atomic_component_id'])
            )
            counts[key] += 1
            first_epoch.setdefault(key, int(epoch))
            last_epoch[key] = int(epoch)

    rows = []
    for item in component_df.sort_values(
        ['subject_id', 'atomic_component_id']
    ).to_dict('records'):
        key = (str(item['subject_id']), int(item['atomic_component_id']))
        tier = str(item['component_tier'])
        eligible = tier in R7_SMALL_COMPONENT_TIERS
        selections = int(counts.get(key, 0)) if eligible else 0
        rows.append({
            'subject_id': key[0],
            'atomic_component_id': key[1],
            'component_tier': tier,
            'component_voxels': int(item['component_voxels']),
            'component_volume_ml': float(item['component_volume_ml']),
            'global_proposal_voxels': int(item['global_proposal_voxels']),
            'local_contrastive_eligible': bool(eligible),
            'epochs_eligible': (
                MAX_EPOCHS - R7_COMPONENT_LOSS_START_EPOCH + 1
                if eligible else 0
            ),
            'selections_epochs_8_to_40': selections,
            'first_selected_epoch': first_epoch.get(key, np.nan),
            'last_selected_epoch': last_epoch.get(key, np.nan),
            'never_selected': bool(eligible and selections == 0),
            'selected_once': bool(eligible and selections == 1),
            'selected_multiple_times': bool(eligible and selections > 1),
        })
    frame = pd.DataFrame(rows)
    summary_rows = []
    for tier in COMPONENT_TIERS:
        group = frame[frame['component_tier'].astype(str) == tier]
        eligible = group[group['local_contrastive_eligible'].astype(bool)]
        values = eligible['selections_epochs_8_to_40'].to_numpy(
            dtype=np.float64
        )
        summary_rows.append({
            'component_tier': tier,
            'components': int(len(group)),
            'eligible_components': int(len(eligible)),
            'never_selected_components': int(
                eligible['never_selected'].sum()
            ),
            'never_selected_fraction': float(
                eligible['never_selected'].mean()
            ) if len(eligible) else 0.0,
            'selected_once_fraction': float(
                eligible['selected_once'].mean()
            ) if len(eligible) else 0.0,
            'minimum_selections': int(values.min()) if len(values) else 0,
            'median_selections': float(np.median(values)) if len(values) else 0.0,
            'mean_selections': float(values.mean()) if len(values) else 0.0,
            'maximum_selections': int(values.max()) if len(values) else 0,
            'selection_gini': r7d_gini(values),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(frame, R7D_EXPOSURE_PATH)
    atomic_write_csv(summary, R7D_EXPOSURE_SUMMARY_PATH)
    return frame, summary


def r7d_maximum_component_cell_matching(cell_sets):
    matched_cell = {}

    def visit(component_index, seen):
        for cell_id in sorted(cell_sets[component_index]):
            if cell_id in seen:
                continue
            seen.add(cell_id)
            if (
                cell_id not in matched_cell
                or visit(matched_cell[cell_id], seen)
            ):
                matched_cell[cell_id] = component_index
                return True
        return False

    matched = 0
    for component_index in range(len(cell_sets)):
        matched += int(visit(component_index, set()))
    return int(matched)


def r7d_resolution_audit():
    component_rows = []
    subject_rows = []
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    canvas_shape = np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    global_shape = tuple(int(v) for v in GLOBAL_INPUT_SHAPE)
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            if partition == 'fold0_fit':
                expected = len(component_records.get(subject_id, []))
                if int(count) != int(expected):
                    raise RuntimeError(
                        f'{subject_id}: resolution audit reconstructed '
                        f'{count} components; frozen audit has {expected}.'
                    )
            pad_start = (canvas_shape - np.asarray(support.shape)) // 2
            cell_sets = []
            component_metadata = []
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for component_id in range(1, int(count) + 1):
                coordinates = np.argwhere(labels == component_id)
                pooled = np.unique(
                    (coordinates + pad_start) // downsample, axis=0
                )
                cell_ids = set(
                    np.ravel_multi_index(pooled.T, global_shape).tolist()
                )
                cell_sets.append(cell_ids)
                voxels = int(len(coordinates))
                volume_ml = voxels * voxel_volume_ml
                component_metadata.append((voxels, volume_ml, pooled))
            cell_frequency = defaultdict(int)
            signature_frequency = defaultdict(int)
            for cells in cell_sets:
                for cell_id in cells:
                    cell_frequency[cell_id] += 1
                signature_frequency[tuple(sorted(cells))] += 1
            matched = r7d_maximum_component_cell_matching(cell_sets)
            collision_components = 0
            duplicate_signature_components = 0
            for component_index, cells in enumerate(cell_sets):
                voxels, volume_ml, pooled = component_metadata[component_index]
                shared_cells = sum(
                    int(cell_frequency[cell_id] > 1) for cell_id in cells
                )
                duplicate_signature = (
                    signature_frequency[tuple(sorted(cells))] > 1
                )
                collision = shared_cells > 0
                collision_components += int(collision)
                duplicate_signature_components += int(duplicate_signature)
                component_rows.append({
                    'development_partition': partition,
                    'subject_id': subject_id,
                    'component_id': int(component_index + 1),
                    'component_tier': pilot_component_tier(volume_ml, voxels),
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'global_proposal_voxels': int(len(cells)),
                    'single_global_voxel': bool(len(cells) == 1),
                    'shared_global_voxels': int(shared_cells),
                    'shares_any_global_voxel': bool(collision),
                    'duplicate_pooled_signature': bool(duplicate_signature),
                    'native_voxels_per_global_voxel': float(
                        voxels / max(len(cells), 1)
                    ),
                })
            subject_rows.append({
                'development_partition': partition,
                'subject_id': subject_id,
                'components': int(count),
                'distinct_global_voxels_used': int(len(cell_frequency)),
                'components_sharing_any_global_voxel': int(collision_components),
                'components_with_duplicate_pooled_signature': int(
                    duplicate_signature_components
                ),
                'maximum_distinct_component_assignment': int(matched),
                'distinct_assignment_ceiling_fraction': float(
                    matched / max(int(count), 1)
                ) if count else 1.0,
            })
            if case_index in {1, len(frame)} or case_index % 20 == 0:
                print(
                    f'  Resolution {partition} {case_index:02d}/{len(frame)}: '
                    f'{subject_id}, components={count}'
                )
    components = pd.DataFrame(component_rows)
    subjects = pd.DataFrame(subject_rows)
    summary_rows = []
    for partition in ['fold0_fit', 'fold0_heldout']:
        for tier in COMPONENT_TIERS:
            group = components[
                (components['development_partition'] == partition)
                & (components['component_tier'] == tier)
            ]
            summary_rows.append({
                'development_partition': partition,
                'component_tier': tier,
                'components': int(len(group)),
                'single_global_voxel_fraction': float(
                    group['single_global_voxel'].mean()
                ) if len(group) else 0.0,
                'global_voxel_collision_fraction': float(
                    group['shares_any_global_voxel'].mean()
                ) if len(group) else 0.0,
                'duplicate_pooled_signature_fraction': float(
                    group['duplicate_pooled_signature'].mean()
                ) if len(group) else 0.0,
                'median_global_proposal_voxels': float(
                    group['global_proposal_voxels'].median()
                ) if len(group) else 0.0,
                'mean_distinct_assignment_ceiling_fraction': float('nan'),
            })
    for partition, group in subjects.groupby('development_partition'):
        summary_rows.append({
            'development_partition': str(partition),
            'component_tier': 'subject_level_distinct_assignment_ceiling',
            'components': int(group['components'].sum()),
            'single_global_voxel_fraction': float('nan'),
            'global_voxel_collision_fraction': float('nan'),
            'duplicate_pooled_signature_fraction': float('nan'),
            'median_global_proposal_voxels': float('nan'),
            'mean_distinct_assignment_ceiling_fraction': float(
                group['distinct_assignment_ceiling_fraction'].mean()
            ),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(components, R7D_RESOLUTION_PATH)
    atomic_write_csv(summary, R7D_RESOLUTION_SUMMARY_PATH)
    return components, subjects, summary


def r7d_validation_artifacts(epoch):
    root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    paths = {
        'case': root / 'case_metrics.csv',
        'component': root / 'component_metrics.csv',
        'tier': root / 'component_tier_metrics.csv',
        'summary': root / 'validation_summary.json',
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise RuntimeError(
            'Completed validation artifacts are missing: ' + ', '.join(missing)
        )
    return paths


def r7d_paired_completed_metrics(r7_best_epoch):
    r5_paths = r7d_validation_artifacts(0)
    r7_paths = r7d_validation_artifacts(r7_best_epoch)
    r5_case = pd.read_csv(r5_paths['case'])
    r7_case = pd.read_csv(r7_paths['case'])
    case_keys = ['subject_id']
    paired_case = r5_case.merge(
        r7_case, on=case_keys, how='outer', suffixes=('_r5', '_r7'),
        validate='one_to_one', indicator=True,
    )
    if not bool((paired_case['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 case metrics do not pair one-to-one.')
    paired_case = paired_case.drop(columns=['_merge'])
    case_delta_columns = [
        'dice', 'voxel_precision', 'voxel_recall',
        'predicted_volume_ml_from_mask', 'predicted_volume_ml_from_router',
        'presence_probability', 'proposal_voxel_precision',
        'proposal_voxel_recall', 'proposal_predicted_to_target_volume_ratio',
    ]
    for name in case_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_case and right in paired_case:
            paired_case[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_case[right], errors='coerce')
                - pd.to_numeric(paired_case[left], errors='coerce')
            )

    r5_component = pd.read_csv(r5_paths['component'])
    r7_component = pd.read_csv(r7_paths['component'])
    component_keys = ['subject_id', 'component_id']
    paired_component = r5_component.merge(
        r7_component, on=component_keys, how='outer',
        suffixes=('_r5', '_r7'), validate='one_to_one', indicator=True,
    )
    if not bool((paired_component['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 component metrics do not pair one-to-one.')
    paired_component = paired_component.drop(columns=['_merge'])
    component_delta_columns = [
        'final_probability_maximum', 'proposal_probability_maximum',
        'final_maximum_component_iou', 'proposal_maximum_component_iou',
        'proposal_minimum_distance_mm',
        'budgeted_proposal_maximum_component_iou',
        'budgeted_proposal_minimum_distance_mm',
    ]
    for name in component_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_component[right], errors='coerce')
                - pd.to_numeric(paired_component[left], errors='coerce')
            )
    for name in [
        'final_detected', 'proposal_covered',
        'budgeted_proposal_within_8mm',
        'representation_failure_below_0p10',
    ]:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                paired_component[right].map(as_bool).astype(int)
                - paired_component[left].map(as_bool).astype(int)
            )
    atomic_write_csv(paired_case, R7D_PAIRED_CASE_PATH)
    atomic_write_csv(paired_component, R7D_PAIRED_COMPONENT_PATH)

    candidates = []
    positive_case_candidates = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    for label, ascending in [
        ('largest_case_dice_regression', True),
        ('largest_case_dice_improvement', False),
    ]:
        ranked = positive_case_candidates.sort_values(
            'dice_delta_r7_minus_r5', ascending=ascending
        ).head(5)
        for rank, item in enumerate(ranked.to_dict('records'), start=1):
            candidates.append({
                'candidate_reason': label,
                'rank': rank,
                'subject_id': str(item['subject_id']),
                'component_id': np.nan,
                'component_tier': '',
                'r5_value': float(item['dice_r5']),
                'r7_value': float(item['dice_r7']),
                'delta_r7_minus_r5': float(
                    item['dice_delta_r7_minus_r5']
                ),
            })
    small = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ].copy()
    small = small.sort_values(
        'proposal_probability_maximum_delta_r7_minus_r5', ascending=True
    ).head(10)
    for rank, item in enumerate(small.to_dict('records'), start=1):
        candidates.append({
            'candidate_reason': 'largest_small_proposal_probability_regression',
            'rank': rank,
            'subject_id': str(item['subject_id']),
            'component_id': int(item['component_id']),
            'component_tier': str(item['component_tier_r7']),
            'r5_value': float(item['proposal_probability_maximum_r5']),
            'r7_value': float(item['proposal_probability_maximum_r7']),
            'delta_r7_minus_r5': float(
                item['proposal_probability_maximum_delta_r7_minus_r5']
            ),
        })
    candidate_frame = pd.DataFrame(candidates)
    atomic_write_csv(candidate_frame, R7D_VISUAL_CANDIDATES_PATH)
    return paired_case, paired_component, candidate_frame


def r7d_checkpoint_pareto(r7_best_epoch):
    history = pd.read_csv(PILOT_HISTORY_PATH).sort_values('epoch').copy()
    maximize = [
        'validation_mean_dice',
        'validation_global_dice',
        'validation_small_mean_dice',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'proposal_voxel_average_precision_256bin',
    ]
    maximize = [name for name in maximize if name in history.columns]
    values = history[maximize].apply(pd.to_numeric, errors='coerce').fillna(
        -np.inf
    ).to_numpy(dtype=np.float64)
    pareto = np.ones(len(history), dtype=bool)
    for index in range(len(history)):
        dominated = np.any(
            np.all(values >= values[index], axis=1)
            & np.any(values > values[index], axis=1)
        )
        pareto[index] = not bool(dominated)
    history['pareto_nondominated'] = pareto
    history['is_selected_best_epoch'] = (
        history['epoch'].astype(int) == int(r7_best_epoch)
    )
    best_rows = {}
    metric_flags = {}
    for name in maximize + ['pilot_checkpoint_score']:
        if name not in history:
            continue
        values_series = pd.to_numeric(history[name], errors='coerce')
        best_index = values_series.idxmax()
        best_epoch = int(history.loc[best_index, 'epoch'])
        best_value = float(values_series.loc[best_index])
        flag = f'is_best_{name}'
        history[flag] = history['epoch'].astype(int) == best_epoch
        best_rows[name] = {'epoch': best_epoch, 'value': best_value}
        metric_flags[name] = flag
    atomic_write_csv(history, R7D_CHECKPOINT_PARETO_PATH)
    return history, best_rows


def r7d_branch_comparison(r5_best_epoch, r7_best_epoch):
    sources = [
        ('r5', R5_PILOT_ROOT / 'branch_reliance_history.csv', r5_best_epoch),
        ('r7', PILOT_BRANCH_HISTORY_PATH, r7_best_epoch),
    ]
    selected_frames = []
    for checkpoint, path, epoch in sources:
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        available_epochs = sorted(frame['epoch'].astype(int).unique().tolist())
        if not available_epochs:
            continue
        selected_epoch = (
            int(epoch) if int(epoch) in available_epochs
            else min(available_epochs, key=lambda value: abs(value - int(epoch)))
        )
        selected = frame[frame['epoch'].astype(int) == selected_epoch].copy()
        selected['checkpoint'] = checkpoint
        selected['requested_best_epoch'] = int(epoch)
        selected['selected_probe_epoch'] = int(selected_epoch)
        selected['exact_best_epoch_probe'] = bool(selected_epoch == int(epoch))
        selected_frames.append(selected)
    if not selected_frames:
        comparison = pd.DataFrame(columns=[
            'checkpoint', 'ablation', 'mean_target_probability_delta'
        ])
    else:
        raw = pd.concat(selected_frames, ignore_index=True)
        rows = []
        for (checkpoint, ablation), group in raw.groupby(
            ['checkpoint', 'ablation']
        ):
            rows.append({
                'checkpoint': str(checkpoint),
                'ablation': str(ablation),
                'subjects': int(group['subject_id'].nunique()),
                'requested_best_epoch': int(
                    group['requested_best_epoch'].iloc[0]
                ),
                'selected_probe_epoch': int(
                    group['selected_probe_epoch'].iloc[0]
                ),
                'exact_best_epoch_probe': bool(
                    group['exact_best_epoch_probe'].iloc[0]
                ),
                'mean_target_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_target_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_local_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_local_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_proposal_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_proposal_probability_delta'],
                    errors='coerce',
                ).mean()),
            })
        comparison = pd.DataFrame(rows)
        if set(comparison['checkpoint']) == {'r5', 'r7'}:
            pivot = comparison.pivot(
                index='ablation', columns='checkpoint',
                values='mean_target_probability_delta',
            )
            deltas = (pivot['r7'] - pivot['r5']).to_dict()
            comparison['target_delta_change_r7_minus_r5'] = comparison[
                'ablation'
            ].map(deltas)
    atomic_write_csv(comparison, R7D_BRANCH_PATH)
    return comparison


def r7d_global_component_masks(labels, count, pad_start):
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pad = np.asarray(pad_start, dtype=np.int64)
    all_coordinates = np.argwhere(labels > 0)
    full = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    if len(all_coordinates):
        pooled = np.unique((all_coordinates + pad) // downsample, axis=0)
        full[tuple(pooled.T)] = True
    components = []
    for component_id in range(1, int(count) + 1):
        coordinates = np.argwhere(labels == component_id)
        pooled = np.unique((coordinates + pad) // downsample, axis=0)
        components.append(pooled)
    return full, components


def r7d_global_domain(native_shape, pad_start):
    start = np.asarray(pad_start, dtype=np.int64)
    end = start + np.asarray(native_shape, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    low_start = start // downsample
    low_end = (end + downsample - 1) // downsample
    domain = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    slices = tuple(
        slice(int(low_start[axis]), int(low_end[axis])) for axis in range(3)
    )
    domain[slices] = True
    return domain


def r7d_component_logit_margins(
    logits, full_target, component_coordinates, domain, subject_id,
    partition, checkpoint, spacing,
):
    rows = []
    outer_radius = np.asarray(R7_OUTER_STRUCTURE.shape, dtype=np.int64) // 2
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    native_voxel_volume = voxel_volume_ml
    for component_id, pooled in enumerate(component_coordinates, start=1):
        if len(pooled) == 0:
            continue
        minimum = np.maximum(pooled.min(axis=0) - outer_radius, 0)
        maximum = np.minimum(
            pooled.max(axis=0) + outer_radius + 1,
            np.asarray(GLOBAL_INPUT_SHAPE),
        )
        slices = tuple(
            slice(int(minimum[axis]), int(maximum[axis]))
            for axis in range(3)
        )
        exact = np.zeros(tuple((maximum - minimum).tolist()), dtype=bool)
        local_coordinates = pooled - minimum
        exact[tuple(local_coordinates.T)] = True
        _, _, negative = r7_component_regions(
            exact, full_target[slices], domain[slices]
        )
        positive_logits = logits[slices][exact]
        negative_logits = logits[slices][negative]
        hard_count = min(
            len(negative_logits),
            max(R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * len(positive_logits)),
        )
        if hard_count > 0:
            hard_negative = np.partition(
                negative_logits, len(negative_logits) - hard_count
            )[-hard_count:]
            hard_negative_mean = float(hard_negative.mean())
            hard_negative_maximum = float(hard_negative.max())
        else:
            hard_negative_mean = float('nan')
            hard_negative_maximum = float('nan')
        native_voxels_estimate = int(len(pooled) * np.prod(GLOBAL_DOWNSAMPLE))
        volume_ml_estimate = native_voxels_estimate * native_voxel_volume
        positive_mean = float(positive_logits.mean())
        positive_minimum = float(positive_logits.min())
        positive_maximum = float(positive_logits.max())
        rows.append({
            'checkpoint': checkpoint,
            'development_partition': partition,
            'subject_id': subject_id,
            'component_id': int(component_id),
            'component_tier_from_pooled_extent': pilot_component_tier(
                volume_ml_estimate, native_voxels_estimate
            ),
            'global_proposal_voxels': int(len(pooled)),
            'positive_logit_mean': positive_mean,
            'positive_logit_minimum': positive_minimum,
            'positive_logit_maximum': positive_maximum,
            'hard_negative_logit_mean': hard_negative_mean,
            'hard_negative_logit_maximum': hard_negative_maximum,
            'positive_minus_hard_negative_mean_margin': (
                positive_mean - hard_negative_mean
                if np.isfinite(hard_negative_mean) else float('nan')
            ),
            'positive_minimum_minus_hard_negative_maximum_margin': (
                positive_minimum - hard_negative_maximum
                if np.isfinite(hard_negative_maximum) else float('nan')
            ),
            'positive_probability_mean': float(
                1.0 / (1.0 + np.exp(-np.clip(positive_mean, -30, 30)))
            ),
            'negative_annulus_voxels': int(negative.sum()),
            'hard_negative_voxels_used': int(hard_count),
        })
    return rows


def r7d_topk_prediction(probability, budget):
    probability = np.asarray(probability, dtype=np.float32)
    budget = int(np.clip(int(budget), 1, probability.size))
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape)


def r7d_budget_metrics(
    probability, target, spacing, predicted_volume_ml, subject_id,
    checkpoint,
):
    target = np.asarray(target, dtype=bool)
    target_voxels = int(target.sum())
    if target_voxels == 0:
        return []
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    labels, count = ndimage.label(
        target, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    component_objects = ndimage.find_objects(labels)
    component_metadata = []
    for component_id, component_slice in enumerate(component_objects, start=1):
        if component_slice is None:
            continue
        local = labels[component_slice] == component_id
        voxels = int(local.sum())
        volume_ml = voxels * voxel_volume_ml
        component_metadata.append((
            component_id, component_slice, local,
            pilot_component_tier(volume_ml, voxels),
        ))
    budgets = [
        ('router_x1', max(1, int(math.ceil(predicted_volume_ml / voxel_volume_ml)))),
        ('router_x2', max(1, int(math.ceil(2.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('router_x3', max(1, int(math.ceil(3.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('oracle_target_x3', max(1, int(math.ceil(3.0 * target_voxels)))),
    ]
    rows = []
    for budget_name, requested_budget in budgets:
        prediction = r7d_topk_prediction(probability, requested_budget)
        predicted_voxels = int(prediction.sum())
        intersection = int((prediction & target).sum())
        distance = ndimage.distance_transform_edt(
            ~prediction, sampling=spacing
        )
        within_8 = []
        small_within_8 = []
        for _, component_slice, component_local, tier in component_metadata:
            minimum_distance = float(
                distance[component_slice][component_local].min(initial=np.inf)
            )
            covered = bool(minimum_distance <= 8.0)
            within_8.append(covered)
            if tier in R7_SMALL_COMPONENT_TIERS:
                small_within_8.append(covered)
        rows.append({
            'checkpoint': checkpoint,
            'subject_id': subject_id,
            'budget_policy': budget_name,
            'router_predicted_volume_ml': float(predicted_volume_ml),
            'target_volume_ml': float(target_voxels * voxel_volume_ml),
            'router_to_target_volume_ratio': float(
                predicted_volume_ml / max(target_voxels * voxel_volume_ml, 1e-8)
            ),
            'requested_budget_voxels': int(requested_budget),
            'selected_budget_voxels': predicted_voxels,
            'selected_to_target_voxel_ratio': float(
                predicted_voxels / max(target_voxels, 1)
            ),
            'voxel_precision': float(intersection / max(predicted_voxels, 1)),
            'voxel_recall': float(intersection / max(target_voxels, 1)),
            'dice': float(
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            ),
            'components': int(len(component_metadata)),
            'component_within_8mm_fraction': float(np.mean(within_8)),
            'small_components': int(len(small_within_8)),
            'small_component_within_8mm_fraction': float(
                np.mean(small_within_8)
            ) if small_within_8 else float('nan'),
        })
    return rows


def r7d_load_model(checkpoint_path, expected_version, expected_revision, device):
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=False
    )
    checks = {
        'pilot_version': checkpoint.get('pilot_version') == expected_version,
        'code_revision': checkpoint.get('code_revision') == expected_revision,
        'parameter_count': int(checkpoint.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS,
        'ema_model_state': 'ema_model_state' in checkpoint,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'Unsafe read-only checkpoint {checkpoint_path}: '
            + ', '.join(failed)
        )
    model = VascuFateNetV22(pilot_schema).to(device)
    model.load_state_dict(checkpoint['ema_model_state'], strict=True)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    del checkpoint
    return model


@torch.no_grad()
def r7d_checkpoint_margin_and_volume_audit(
    checkpoint_label, checkpoint_path, expected_version,
    expected_revision, device,
):
    model = r7d_load_model(
        checkpoint_path, expected_version, expected_revision, device
    )
    margin_rows = []
    budget_rows = []
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            case = pilot_load_validation_case(row)
            batch = pilot_stack_sample(case['base_sample'], device)
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(model, batch)
            proposal_logits_tensor = cache['proposal_logits'].float()
            logits = proposal_logits_tensor[0, 0].cpu().numpy()
            predicted_volume_ml = float(torch.expm1(
                cache['routed']['log_volume_prediction'].float()
            )[0].clamp_min(0).cpu())
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            full_global, component_coordinates = r7d_global_component_masks(
                labels, count, case['pad_start']
            )
            domain = r7d_global_domain(case['native_shape'], case['pad_start'])
            rows = r7d_component_logit_margins(
                logits, full_global, component_coordinates, domain,
                subject_id, partition, checkpoint_label, spacing,
            )
            
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for item in rows:
                component_id = int(item['component_id'])
                voxels = int((labels == component_id).sum())
                volume_ml = voxels * voxel_volume_ml
                item['component_voxels'] = voxels
                item['component_volume_ml'] = volume_ml
                item['component_tier'] = pilot_component_tier(
                    volume_ml, voxels
                )
            margin_rows.extend(rows)

            if (
                partition == 'fold0_heldout'
                and as_bool(row['trial8_hard_metric_eligible'])
                and bool(case['target'].any())
            ):
                proposal_canvas = F.interpolate(
                    torch.sigmoid(proposal_logits_tensor),
                    size=GLOBAL_CANVAS_SHAPE,
                    mode='trilinear', align_corners=False,
                )[0, 0].cpu().numpy()
                native_slices = tuple(
                    slice(
                        case['pad_start'][axis],
                        case['pad_start'][axis] + case['native_shape'][axis],
                    )
                    for axis in range(3)
                )
                proposal_native = proposal_canvas[native_slices]
                budget_rows.extend(r7d_budget_metrics(
                    proposal_native, case['target'], case['spacing'],
                    predicted_volume_ml, subject_id, checkpoint_label,
                ))
            del batch, cache, proposal_logits_tensor, case
            if case_index in {1, len(frame)} or case_index % 10 == 0:
                print(
                    f'  {checkpoint_label} {partition} '
                    f'{case_index:02d}/{len(frame)}: {subject_id}'
                )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return margin_rows, budget_rows


def r7d_margin_summary(frame):
    rows = []
    for (checkpoint, partition, tier), group in frame.groupby([
        'checkpoint', 'development_partition', 'component_tier'
    ]):
        margin = pd.to_numeric(
            group['positive_minus_hard_negative_mean_margin'],
            errors='coerce',
        )
        rows.append({
            'checkpoint': str(checkpoint),
            'development_partition': str(partition),
            'component_tier': str(tier),
            'components': int(len(group)),
            'mean_logit_margin': float(margin.mean()),
            'median_logit_margin': float(margin.median()),
            'positive_margin_fraction': float((margin > 0).mean()),
            'mean_positive_probability': float(pd.to_numeric(
                group['positive_probability_mean'], errors='coerce'
            ).mean()),
        })
    return pd.DataFrame(rows)


def r7d_volume_budget_summary(frame):
    rows = []
    for (checkpoint, policy), group in frame.groupby(
        ['checkpoint', 'budget_policy']
    ):
        rows.append({
            'checkpoint': str(checkpoint),
            'budget_policy': str(policy),
            'subjects': int(group['subject_id'].nunique()),
            'mean_dice': float(group['dice'].mean()),
            'mean_voxel_precision': float(group['voxel_precision'].mean()),
            'mean_voxel_recall': float(group['voxel_recall'].mean()),
            'mean_component_within_8mm_fraction': float(
                group['component_within_8mm_fraction'].mean()
            ),
            'mean_small_component_within_8mm_fraction': float(
                group['small_component_within_8mm_fraction'].mean()
            ),
            'median_router_to_target_volume_ratio': float(
                group['router_to_target_volume_ratio'].median()
            ),
            'mean_selected_to_target_voxel_ratio': float(
                group['selected_to_target_voxel_ratio'].mean()
            ),
        })
    return pd.DataFrame(rows)


def r7d_small_margin_delta(summary, partition):
    subset = summary[
        (summary['development_partition'] == partition)
        & summary['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    if not len(subset):
        return float('nan')
    weighted = {}
    for checkpoint in ['r5', 'r7']:
        group = subset[subset['checkpoint'] == checkpoint]
        if not len(group):
            weighted[checkpoint] = float('nan')
            continue
        weighted[checkpoint] = float(np.average(
            group['mean_logit_margin'], weights=group['components']
        ))
    return weighted['r7'] - weighted['r5']


def run_r7_failure_mechanism_diagnostic():
    if r7d_existing_result_is_complete():
        gate = load_json(R7_FAILURE_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC ALREADY COMPLETE')
        print('=' * 80)
        print(f'Dominant mechanism: {gate["dominant_failure_mechanism"]}')
        print(f'Diagnostic gate:    {R7_FAILURE_GATE_PATH}')
        print('No source checkpoint or completed diagnostic was rewritten.')
        return

    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r7_best_epoch = int(r7_result['best_epoch'])
    r5_best_epoch = int(r5_result['best_epoch'])
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_sha_before = file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    source_paths = [
        R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH,
        R5_RESULT_GATE_PATH, PILOT_RESULT_GATE_PATH, PILOT_HISTORY_PATH,
    ]
    source_hashes_before = {
        str(path): file_sha256(path) for path in source_paths
    }
    initial_checks = {
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r7_pilot_complete': r7_result.get('pilot_complete') is True,
        'r5_checkpoint_hash_matches_result_gate': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'r7_checkpoint_hash_matches_result_gate': (
            r7_result.get('best_checkpoint_sha256') == r7_sha_before
        ),
        'r7_directional_success_was_false': (
            r7_result.get('r7_directional_success_passed') is False
        ),
        'full_fivefold_training_locked': (
            r7_result.get('full_fivefold_training_authorized') is False
        ),
        'fold0_subject_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
        ),
        'sealed_test_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
        ),
    }
    if not all(initial_checks.values()):
        failed = [name for name, passed in initial_checks.items() if not passed]
        raise RuntimeError(
            'R7 failure diagnostic upstream checks failed: '
            + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC STARTING')
    print('=' * 80)
    print(f'R5 / r7 best epochs:               {r5_best_epoch} / {r7_best_epoch}')
    print('Target access:                     fold-0 development only')
    print('Locked validation targets read:    False')
    print('Test targets read:                 False')
    print('Optimizer / backward / steps:      0 / 0 / 0')

    target_access_frame = pd.DataFrame([
        {
            'subject_id': str(row['subject_id']),
            'development_partition': partition,
            'target_arrays_authorized': True,
            'target_arrays_read_by_diagnostic': True,
            'locked_validation_target_read': False,
            'sealed_test_target_read': False,
        }
        for partition, frame in [
            ('fold0_fit', fold0_fit_df),
            ('fold0_heldout', fold0_heldout_df),
        ]
        for row in frame.to_dict('records')
    ])
    atomic_write_csv(target_access_frame, R7D_TARGET_ACCESS_PATH)

    exposure, exposure_summary = r7d_component_schedule_exposure()
    resolution, resolution_subjects, resolution_summary = (
        r7d_resolution_audit()
    )
    paired_case, paired_component, visual_candidates = (
        r7d_paired_completed_metrics(r7_best_epoch)
    )
    checkpoint_pareto, metric_best_rows = r7d_checkpoint_pareto(
        r7_best_epoch
    )
    branch_comparison = r7d_branch_comparison(
        r5_best_epoch, r7_best_epoch
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            'The train-versus-held-out r5/r7 margin audit requires CUDA.'
        )
    set_determinism(PILOT_SEED)
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    all_margin_rows = []
    all_budget_rows = []
    checkpoint_specs = [
        ('r5', R5_BEST_CHECKPOINT_PATH,
         R5_PILOT_VERSION, R5_PILOT_CODE_REVISION),
        ('r7', PILOT_BEST_CHECKPOINT_PATH,
         PILOT_VERSION, PILOT_CODE_REVISION),
    ]
    for label, path, version, revision in checkpoint_specs:
        print(f'\nRead-only global proposal audit: {label}')
        margin_rows, budget_rows = r7d_checkpoint_margin_and_volume_audit(
            label, path, version, revision, device
        )
        all_margin_rows.extend(margin_rows)
        all_budget_rows.extend(budget_rows)
    margins = pd.DataFrame(all_margin_rows)
    budgets = pd.DataFrame(all_budget_rows)
    margin_summary = r7d_margin_summary(margins)
    budget_summary = r7d_volume_budget_summary(budgets)
    atomic_write_csv(margins, R7D_MARGIN_PATH)
    atomic_write_csv(margin_summary, R7D_MARGIN_SUMMARY_PATH)
    atomic_write_csv(budgets, R7D_VOLUME_BUDGET_PATH)
    atomic_write_csv(budget_summary, R7D_VOLUME_BUDGET_SUMMARY_PATH)

    small_exposure = exposure[
        exposure['local_contrastive_eligible'].astype(bool)
    ]
    never_selected_fraction = float(
        small_exposure['never_selected'].mean()
    )
    heldout_small_resolution = resolution[
        (resolution['development_partition'] == 'fold0_heldout')
        & resolution['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    heldout_small_collision_fraction = float(
        heldout_small_resolution['shares_any_global_voxel'].mean()
    ) if len(heldout_small_resolution) else float('nan')
    heldout_assignment_ceiling = float(
        resolution_subjects[
            resolution_subjects['development_partition'] == 'fold0_heldout'
        ]['distinct_assignment_ceiling_fraction'].mean()
    )
    fit_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_fit'
    )
    heldout_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_heldout'
    )
    positive_paired_cases = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    case_dice_delta = float(
        positive_paired_cases['dice_delta_r7_minus_r5'].mean()
    )
    small_components = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ]
    small_proposal_probability_delta = float(pd.to_numeric(
        small_components[
            'proposal_probability_maximum_delta_r7_minus_r5'
        ], errors='coerce'
    ).mean())
    r7_budget = budget_summary[
        budget_summary['checkpoint'] == 'r7'
    ]
    router_x3 = r7_budget[
        r7_budget['budget_policy'] == 'router_x3'
    ]
    oracle_x3 = r7_budget[
        r7_budget['budget_policy'] == 'oracle_target_x3'
    ]
    router_small_within8 = float(
        router_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    oracle_small_within8 = float(
        oracle_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    deployment_budget_gap = oracle_small_within8 - router_small_within8
    selected_epoch = int(r7_best_epoch)
    mean_dice_best_epoch = int(
        metric_best_rows['validation_mean_dice']['epoch']
    )
    checkpoint_selection_mismatch = selected_epoch != mean_dice_best_epoch

    mechanism_evidence = {
        'component_exposure_imbalance': bool(never_selected_fraction > 0.10),
        'low_resolution_component_collision': bool(
            heldout_small_collision_fraction > 0.10
            or heldout_assignment_ceiling < 0.95
        ),
        'contrastive_train_heldout_margin_divergence': bool(
            np.isfinite(fit_small_margin_delta)
            and np.isfinite(heldout_small_margin_delta)
            and fit_small_margin_delta > heldout_small_margin_delta + 0.10
        ),
        'small_proposal_signal_regressed': bool(
            small_proposal_probability_delta < 0.0
        ),
        'deployment_volume_router_limits_coverage': bool(
            deployment_budget_gap > 0.10
        ),
        'checkpoint_objective_mismatch': bool(checkpoint_selection_mismatch),
        'r7_mean_dice_regressed': bool(case_dice_delta < 0.0),
        'r7_representation_failure_extreme': bool(
            float(r7_result['best_validation_summary'][
                'small_component_representation_failure_fraction'
            ]) > 0.75
        ),
    }
    if mechanism_evidence['contrastive_train_heldout_margin_divergence']:
        dominant_mechanism = (
            'low-resolution local-contrastive over-specialization with '
            'poor held-out satellite transfer'
        )
    elif mechanism_evidence['component_exposure_imbalance']:
        dominant_mechanism = (
            'incomplete atomic-component exposure in the stochastic curriculum'
        )
    elif mechanism_evidence['low_resolution_component_collision']:
        dominant_mechanism = (
            'global proposal-grid resolution and component-collision ceiling'
        )
    else:
        dominant_mechanism = (
            'proposal supervision/refinement mismatch rather than missing '
            'clinical or arterial routing capacity'
        )
    recommended_r8 = {
        'architecture': (
            'Preserve the r5 V2.2 global/arterial/clinical path; add a bounded '
            'native-resolution local satellite-proposal head and fuse it as '
            'a zero-initialized residual, leaving the coarse global proposal '
            'responsible for burden and broad localization.'
        ),
        'curriculum': (
            'Replace stochastic inverse-size component draws with a '
            'coverage-guaranteed component cycle before repeats, stratified '
            'by subvoxel/micro/tiny/small tier and subject.'
        ),
        'objective': (
            'Apply component contrast at native local resolution, normalize '
            'per component, cap its gradient contribution against the main '
            'segmentation loss, and retain the coarse proposal loss.'
        ),
        'deployment': (
            'Train and calibrate the predicted-volume router on fold-fit data '
            'only; use predicted rather than target volume for any inference '
            'budget and report the oracle budget only as a diagnostic ceiling.'
        ),
        'selection': (
            'Use a Pareto-safe checkpoint rule that requires no mean-Dice '
            'regression and explicit held-out small-component improvement; '
            'do not optimize the present composite score alone.'
        ),
        'next_stage': (
            'Build an R8 read-only architecture/loss preflight first, then one '
            'fold-0 pilot. Five-fold and locked validation/test evaluation '
            'remain unauthorized.'
        ),
    }

    source_hashes_after = {
        str(path): file_sha256(path) for path in source_paths
    }
    final_checks = {
        **initial_checks,
        'fit_component_count_matches_frozen_872': int(
            (resolution['development_partition'] == 'fold0_fit').sum()
        ) == 872,
        'exposure_covers_every_frozen_fit_component': len(exposure)
        == len(component_df),
        'paired_case_metrics_have_20_subjects': len(paired_case) == 20,
        'paired_component_metrics_are_nonempty': len(paired_component) > 0,
        'margin_audit_has_both_checkpoints': set(margins['checkpoint'])
        == {'r5', 'r7'},
        'margin_audit_has_fit_and_heldout': set(
            margins['development_partition']
        ) == {'fold0_fit', 'fold0_heldout'},
        'all_finite_margin_inputs': bool(np.isfinite(pd.to_numeric(
            margins['positive_logit_mean'], errors='coerce'
        )).all()),
        'deployment_budget_uses_heldout_development_only': bool(
            budgets['subject_id'].isin(fold0_heldout_df['subject_id']).all()
        ),
        'development_target_access_has_exactly_102_subjects': (
            len(target_access_frame) == 102
            and target_access_frame['subject_id'].nunique() == 102
        ),
        'development_target_access_excludes_locked_splits': bool(
            (~target_access_frame['locked_validation_target_read']).all()
            and (~target_access_frame['sealed_test_target_read']).all()
        ),
        'r5_checkpoint_unchanged': r5_sha_before
        == file_sha256(R5_BEST_CHECKPOINT_PATH),
        'r7_checkpoint_unchanged': r7_sha_before
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'all_source_artifacts_unchanged': source_hashes_before
        == source_hashes_after,
        'optimizer_not_constructed': True,
        'backward_not_called': True,
        'optimizer_steps_are_zero': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    diagnostic_passed = all(final_checks.values())

    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': R7_FAILURE_DIAGNOSTIC_VERSION,
        'code_revision': R7_FAILURE_DIAGNOSTIC_CODE_REVISION,
        'r5_best_epoch': r5_best_epoch,
        'r7_best_epoch': r7_best_epoch,
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_sha256': r7_sha_before,
        'observed_effects': {
            'mean_paired_case_dice_delta_r7_minus_r5': case_dice_delta,
            'small_proposal_probability_delta_r7_minus_r5': (
                small_proposal_probability_delta
            ),
            'eligible_components_never_selected_fraction': (
                never_selected_fraction
            ),
            'heldout_small_global_voxel_collision_fraction': (
                heldout_small_collision_fraction
            ),
            'heldout_distinct_assignment_ceiling_fraction': (
                heldout_assignment_ceiling
            ),
            'fit_small_logit_margin_delta_r7_minus_r5': (
                fit_small_margin_delta
            ),
            'heldout_small_logit_margin_delta_r7_minus_r5': (
                heldout_small_margin_delta
            ),
            'r7_router_x3_small_within_8mm_fraction': router_small_within8,
            'r7_oracle_x3_small_within_8mm_fraction': oracle_small_within8,
            'deployment_to_oracle_small_within_8mm_gap': (
                deployment_budget_gap
            ),
            'selected_checkpoint_epoch': selected_epoch,
            'mean_dice_best_epoch': mean_dice_best_epoch,
        },
        'mechanism_evidence': mechanism_evidence,
        'dominant_failure_mechanism': dominant_mechanism,
        'recommended_r8_contract': recommended_r8,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': False,
        'backward_called': False,
        'optimizer_steps': 0,
        'r8_preflight_implementation_allowed': bool(diagnostic_passed),
        'r8_training_authorized': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(summary, R7_FAILURE_SUMMARY_PATH)
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(value)}
        for name, value in final_checks.items()
    ])
    atomic_write_csv(checks_frame, R7D_CHECKS_PATH)
    output_hashes = {
        path.name: file_sha256(path) for path in R7D_OUTPUT_PATHS
    }
    gate = {
        **summary,
        'diagnostic_complete': bool(diagnostic_passed),
        'diagnostic_passed': bool(diagnostic_passed),
        'checks': final_checks,
        'source_sha256_before': source_hashes_before,
        'source_sha256_after': source_hashes_after,
        'output_sha256': output_hashes,
        'diagnostic_root': str(R7_FAILURE_DIAGNOSTIC_ROOT),
        'summary_path': str(R7_FAILURE_SUMMARY_PATH),
    }
    atomic_write_json(gate, R7_FAILURE_GATE_PATH)
    if not diagnostic_passed:
        failed = [
            name for name, passed in final_checks.items() if not passed
        ]
        raise RuntimeError(
            'TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC FAILED. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nComponent exposure by tier:')
    display(exposure_summary)
    print('\nProposal-grid resolution summary:')
    display(resolution_summary)
    print('\nTrain-versus-held-out proposal-logit margins:')
    display(margin_summary)
    print('\nDeployment-legal predicted-volume budget audit:')
    display(budget_summary)
    print('\nBranch reliance comparison:')
    display(branch_comparison)
    print('\nDiagnostic checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Dominant mechanism:                 {dominant_mechanism}')
    print(f'Eligible components never selected: {never_selected_fraction:.4f}')
    print(f'Fit small-margin change r7-r5:      {fit_small_margin_delta:+.4f}')
    print(f'Heldout small-margin change r7-r5:  {heldout_small_margin_delta:+.4f}')
    print(f'R7 router-x3 small within 8 mm:     {router_small_within8:.4f}')
    print(f'R7 oracle-x3 small within 8 mm:     {oracle_small_within8:.4f}')
    print(f'Mean-Dice best / selected epoch:    {mean_dice_best_epoch} / {selected_epoch}')
    print('Optimizer / backward / steps:       0 / 0 / 0')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('R8 preflight implementation allowed: True')
    print('R8 training authorized:             False')
    print('Full five-fold training authorized: False')
    print(f'Diagnostic summary:                 {R7_FAILURE_SUMMARY_PATH}')
    print(f'Diagnostic gate:                    {R7_FAILURE_GATE_PATH}')
    print(
        '\nNext: construct the read-only R8 dual-resolution satellite-proposal '
        'and component-cycled curriculum preflight; do not train yet.'
    )


run_r7_failure_mechanism_diagnostic()






R8_MODEL_VERSION = 'trial8_vascufate_net_v2_3_dual_resolution_satellite'
R8_PREFLIGHT_VERSION = 'v1_r5_warmstart_component_cycle_native_satellite'
R8_CODE_REVISION = (
    'r1_native_component_absolute_calibration_pareto_guard'
)
R8_SEED = 20261007
R8_SATELLITE_GATE_BOUND = 0.20
R8_SATELLITE_FEATURE_CHANNELS = 16
R8_COMPONENT_POSITIVE_PATCHES = 112
R8_TOPOLOGY_NEGATIVE_PATCHES = 28
R8_BRAIN_NEGATIVE_PATCHES = 28
R8_SCHEDULE_EPOCHS = 40
R8_NATIVE_INNER_RADIUS_MM = 2.0
R8_NATIVE_OUTER_RADIUS_MM = 12.0
R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR = 1.0
R8_MAXIMUM_NEW_PARAMETERS = 250_000
R8_TIER_QUOTAS = {
    'subvoxel_or_single_voxel': 2,
    'micro': 76,
    'tiny': 27,
    'small': 7,
}

R8_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / R8_MODEL_VERSION / R8_PREFLIGHT_VERSION
)
R8_PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
R8_ARCHITECTURE_PATH = R8_PREFLIGHT_ROOT / 'r8_architecture_config.json'
R8_TRAINING_POLICY_PATH = R8_PREFLIGHT_ROOT / 'r8_training_policy.json'
R8_SCHEDULE_PATH = R8_PREFLIGHT_ROOT / 'r8_component_cycle_schedule.csv'
R8_SCHEDULE_SUMMARY_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_component_cycle_summary.csv'
)
R8_NATIVE_TARGET_AUDIT_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_native_component_target_audit.csv'
)
R8_PARAMETER_PATH = R8_PREFLIGHT_ROOT / 'r8_parameter_audit.csv'
R8_OUTPUT_PATH = R8_PREFLIGHT_ROOT / 'r8_output_shape_audit.csv'
R8_NEUTRALITY_PATH = R8_PREFLIGHT_ROOT / 'r8_neutrality_audit.csv'
R8_GRADIENT_PATH = R8_PREFLIGHT_ROOT / 'r8_gradient_audit.csv'
R8_LOSS_PATH = R8_PREFLIGHT_ROOT / 'r8_loss_smoke_audit.csv'
R8_CHECKS_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_checks.csv'
R8_GATE_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_gate.json'

R8_OUTPUT_ARTIFACTS = [
    R8_ARCHITECTURE_PATH,
    R8_TRAINING_POLICY_PATH,
    R8_SCHEDULE_PATH,
    R8_SCHEDULE_SUMMARY_PATH,
    R8_NATIVE_TARGET_AUDIT_PATH,
    R8_PARAMETER_PATH,
    R8_OUTPUT_PATH,
    R8_NEUTRALITY_PATH,
    R8_GRADIENT_PATH,
    R8_LOSS_PATH,
    R8_CHECKS_PATH,
]


def r8_existing_preflight_complete():
    if not R8_GATE_PATH.exists():
        return False
    gate = load_json(R8_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('model_version') == R8_MODEL_VERSION
        and gate.get('preflight_version') == R8_PREFLIGHT_VERSION
        and gate.get('code_revision') == R8_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_failure_diagnostic_sha256')
        == file_sha256(R7_FAILURE_GATE_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R8_OUTPUT_ARTIFACTS
    )


def r8_tier_records():
    records = {}
    for tier in R8_TIER_QUOTAS:
        frame = component_df[
            component_df['component_tier'].astype(str) == tier
        ].sort_values(['subject_id', 'atomic_component_id'])
        records[tier] = frame.to_dict('records')
        if not records[tier]:
            raise RuntimeError(f'R8 component tier {tier!r} is empty.')
    return records


R8_COMPONENT_RECORDS_BY_TIER = r8_tier_records()


def r8_tier_permutation(tier, cycle_index):
    tier_index = list(R8_TIER_QUOTAS).index(str(tier))
    rng = np.random.default_rng(
        R8_SEED + tier_index * 10_000_019 + int(cycle_index) * 1_000_003
    )
    count = len(R8_COMPONENT_RECORDS_BY_TIER[str(tier)])
    return rng.permutation(count).tolist()


def r8_component_for_position(tier, position):
    records = R8_COMPONENT_RECORDS_BY_TIER[str(tier)]
    cycle_index = int(position) // len(records)
    offset = int(position) % len(records)
    permutation = r8_tier_permutation(tier, cycle_index)
    item = dict(records[int(permutation[offset])])
    item['r8_tier_cycle_index'] = int(cycle_index)
    item['r8_tier_cycle_offset'] = int(offset)
    return item


def r8_build_epoch_schedule(epoch):
    epoch = int(epoch)
    if epoch < 1:
        raise ValueError('R8 epoch must be positive.')
    positives = []
    for tier, quota in R8_TIER_QUOTAS.items():
        for slot in range(int(quota)):
            position = (epoch - 1) * int(quota) + slot
            component = r8_component_for_position(tier, position)
            subject_id = str(component['subject_id'])
            jitter_rng = np.random.default_rng(
                R8_SEED + epoch * 10_000_019
                + list(R8_TIER_QUOTAS).index(tier) * 104_729 + slot
            )
            center = np.asarray([
                component['centroid_x_vox'],
                component['centroid_y_vox'],
                component['centroid_z_vox'],
            ], dtype=np.float64)
            jitter = np.asarray([
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-3, 4),
            ])
            shape = np.asarray(
                pilot_mask_records[subject_id]['shape'], dtype=np.int64
            )
            center = np.clip(np.rint(center).astype(np.int64) + jitter, 0, shape - 1)
            positives.append({
                'epoch': epoch,
                'sample_type': 'positive',
                'subject_id': subject_id,
                'center_native': tuple(int(value) for value in center),
                'atomic_component_id': int(
                    component['atomic_component_id']
                ),
                'component_tier': tier,
                'component_volume_ml': float(
                    component['component_volume_ml']
                ),
                'r8_tier_cycle_index': int(
                    component['r8_tier_cycle_index']
                ),
                'r8_tier_cycle_offset': int(
                    component['r8_tier_cycle_offset']
                ),
            })
    if len(positives) != R8_COMPONENT_POSITIVE_PATCHES:
        raise RuntimeError(
            f'R8 epoch {epoch} has {len(positives)} positive patches.'
        )
    inherited = pilot_build_epoch_schedule(epoch)
    negatives = [
        dict(item) for item in inherited
        if str(item['sample_type']) != 'positive'
    ]
    if sum(str(item['sample_type']) == 'topology_negative' for item in negatives) != 28:
        raise RuntimeError('R8 topology-negative count changed.')
    if sum(str(item['sample_type']) == 'brain_negative' for item in negatives) != 28:
        raise RuntimeError('R8 brain-negative count changed.')
    for item in negatives:
        item['epoch'] = epoch
        item['r8_tier_cycle_index'] = -1
        item['r8_tier_cycle_offset'] = -1
    schedule = positives + negatives
    rng = np.random.default_rng(R8_SEED + epoch * 1_299_721)
    rng.shuffle(schedule)
    if len(schedule) != 168:
        raise RuntimeError(f'R8 epoch {epoch} has {len(schedule)} samples.')
    return schedule


def r8_schedule_audit():
    schedules = []
    repeated = []
    for epoch in range(1, R8_SCHEDULE_EPOCHS + 1):
        first = r8_build_epoch_schedule(epoch)
        second = r8_build_epoch_schedule(epoch)
        if first != second:
            raise RuntimeError(f'R8 epoch {epoch} schedule is not deterministic.')
        schedules.extend(first)
        repeated.extend(second)
    frame = pd.DataFrame(schedules)
    positive = frame[frame['sample_type'].astype(str) == 'positive'].copy()
    positive['component_key'] = (
        positive['subject_id'].astype(str) + ':'
        + positive['atomic_component_id'].astype(int).astype(str)
    )
    expected_keys = set(
        component_df[
            component_df['component_tier'].astype(str).isin(R8_TIER_QUOTAS)
        ].assign(component_key=lambda value:
            value['subject_id'].astype(str) + ':'
            + value['atomic_component_id'].astype(int).astype(str)
        )['component_key']
    )
    observed_keys = set(positive['component_key'])
    rows = []
    for tier, quota in R8_TIER_QUOTAS.items():
        tier_positive = positive[
            positive['component_tier'].astype(str) == tier
        ]
        counts = tier_positive.groupby('component_key').size()
        records = R8_COMPONENT_RECORDS_BY_TIER[tier]
        completed_cycles_are_unique = True
        for cycle_index, cycle in tier_positive.groupby('r8_tier_cycle_index'):
            keys = cycle['component_key'].tolist()
            if len(cycle) == len(records) and len(keys) != len(set(keys)):
                completed_cycles_are_unique = False
        rows.append({
            'component_tier': tier,
            'tier_components': len(records),
            'quota_per_epoch': int(quota),
            'total_selections_40_epochs': int(len(tier_positive)),
            'observed_components': int(counts.size),
            'never_selected_components': int(len(records) - counts.size),
            'minimum_selections': int(counts.min()),
            'median_selections': float(counts.median()),
            'mean_selections': float(counts.mean()),
            'maximum_selections': int(counts.max()),
            'selection_gini': r7d_gini(counts.to_numpy()),
            'maximum_minus_minimum_selections': int(
                counts.max() - counts.min()
            ),
            'complete_cycles_have_no_duplicates': bool(
                completed_cycles_are_unique
            ),
        })
    summary = pd.DataFrame(rows)
    per_epoch = positive.groupby('epoch').agg(
        positive_patches=('component_key', 'size'),
        unique_components=('component_key', 'nunique'),
        unique_subjects=('subject_id', 'nunique'),
    ).reset_index()
    summary.attrs['all_eligible_components_covered'] = (
        observed_keys == expected_keys
    )
    summary.attrs['minimum_unique_positive_subjects_per_epoch'] = int(
        per_epoch['unique_subjects'].min()
    )
    atomic_write_csv(frame, R8_SCHEDULE_PATH)
    atomic_write_csv(summary, R8_SCHEDULE_SUMMARY_PATH)
    return frame, summary, per_epoch, expected_keys, observed_keys


def r8_native_component_targets(row, local_origin, component_id):
    subject_id = str(row['subject_id'])
    binary, soft, support, spacing = r7d_component_support(row)
    labels, count = ndimage.label(
        support, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    expected = len(component_records.get(subject_id, []))
    if int(count) != int(expected):
        raise RuntimeError(
            f'{subject_id}: R8 reconstructed {count} components; expected {expected}.'
        )
    component_id = int(component_id)
    if not 1 <= component_id <= count:
        raise RuntimeError(f'{subject_id}: invalid component {component_id}.')
    selected = labels == component_id
    selected_canvas, start_a = symmetric_pad(
        selected.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    support_canvas, start_b = symmetric_pad(
        support.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(support, dtype=np.uint8),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if not (start_a == start_b == start_c):
        raise RuntimeError(f'{subject_id}: R8 target padding disagreement.')
    exact = crop_at(
        selected_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    full = crop_at(
        support_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    domain = crop_at(
        domain_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    if not bool(exact.any()):
        raise RuntimeError(f'{subject_id}: R8 native exact component is empty.')
    distance = ndimage.distance_transform_edt(~exact, sampling=spacing)
    negative = (
        (distance >= R8_NATIVE_INNER_RADIUS_MM)
        & (distance <= R8_NATIVE_OUTER_RADIUS_MM)
        & domain & ~full
    )
    if not bool(negative.any()):
        raise RuntimeError(f'{subject_id}: R8 native annulus is empty.')
    retained_fraction = float(exact.sum() / max(selected.sum(), 1))
    return {
        'native_component_target': exact[None],
        'native_component_negative_annulus': negative[None],
        'spacing': spacing,
        'component_native_voxels': int(selected.sum()),
        'component_patch_voxels': int(exact.sum()),
        'component_retained_fraction': retained_fraction,
        'negative_annulus_voxels': int(negative.sum()),
        'binary': binary,
        'soft': soft,
    }


def r8_native_target_probe(schedule_frame):
    probes = []
    positive = schedule_frame[
        schedule_frame['sample_type'].astype(str) == 'positive'
    ]
    for tier in R8_TIER_QUOTAS:
        candidates = positive[
            positive['component_tier'].astype(str) == tier
        ].head(8)
        for item in candidates.to_dict('records'):
            subject_id = str(item['subject_id'])
            row = fit_row_by_subject[subject_id]
            local_origin = pilot_origin_from_center(
                item['center_native'], pilot_mask_records[subject_id]['shape']
            )
            target = r8_native_component_targets(
                row, local_origin, item['atomic_component_id']
            )
            probes.append({
                'subject_id': subject_id,
                'component_tier': tier,
                'atomic_component_id': int(item['atomic_component_id']),
                'component_native_voxels': target['component_native_voxels'],
                'component_patch_voxels': target['component_patch_voxels'],
                'component_retained_fraction': target[
                    'component_retained_fraction'
                ],
                'negative_annulus_voxels': target[
                    'negative_annulus_voxels'
                ],
                'exact_and_annulus_disjoint': not bool(np.any(
                    target['native_component_target']
                    & target['native_component_negative_annulus']
                )),
                'target_arrays_read_split': 'fold0_fit',
                'locked_validation_target_read': False,
                'test_target_read': False,
            })
    frame = pd.DataFrame(probes)
    atomic_write_csv(frame, R8_NATIVE_TARGET_AUDIT_PATH)
    return frame


class NativeSatelliteProposalHead(nn.Module):
    def __init__(self, local_channels=32, hidden=R8_SATELLITE_FEATURE_CHANNELS):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(local_channels + 2, hidden, 1, bias=False),
            nn.GroupNorm(group_count(hidden), hidden), nn.GELU(),
        )
        self.refine = ResidualBlock3D(hidden, hidden, dropout=0.05)
        self.output = nn.Conv3d(hidden, 1, 1)

    def forward(self, local_feature, coarse_proposal, arterial_local):
        feature = self.project(torch.cat([
            local_feature, torch.sigmoid(coarse_proposal), arterial_local,
        ], dim=1))
        return self.output(self.refine(feature))


class VascuFateNetR8(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.base = VascuFateNetV22(schema)
        self.satellite_head = NativeSatelliteProposalHead()
        self.satellite_gate = ConfidenceGate(
            confidence_dim=5, output_dim=1,
            bound=R8_SATELLITE_GATE_BOUND, hidden_dim=32,
        )
        self._decoder_feature = None
        self._decoder_hook = self.base.local_model.decoder1.register_forward_hook(
            self._capture_decoder_feature
        )
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def _capture_decoder_feature(self, module, inputs, output):
        self._decoder_feature = output

    def load_r5_base_state(self, state):
        self.base.load_state_dict(state, strict=True)

    def activate_satellite_gate_for_gradient_test(self, fraction=0.5):
        self.satellite_gate.activate_for_gradient_test(fraction)

    def reset_satellite_gate_to_zero(self):
        with torch.no_grad():
            self.satellite_gate.network[-1].weight.zero_()
            self.satellite_gate.network[-1].bias.zero_()

    def forward(self, **kwargs):
        self._decoder_feature = None
        output = self.base(**kwargs)
        if self._decoder_feature is None:
            raise RuntimeError('R8 decoder feature hook did not fire.')
        arterial_local = sample_global_to_local(
            output['arterial_global_map'][:, None],
            kwargs['local_origin_in_canvas'],
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        if bool(kwargs.get('force_global_dropout', False)):
            coarse = torch.zeros_like(output['global_proposal_local'])
            arterial_local = torch.zeros_like(arterial_local)
        else:
            coarse = output['global_proposal_local']
        satellite_logits = self.satellite_head(
            self._decoder_feature, coarse, arterial_local
        )
        confidence = torch.cat([
            kwargs['confidence_features'],
            torch.sigmoid(output['lesion_presence_logit'])[:, None],
        ], dim=-1)
        gate = self.satellite_gate(confidence)
        base_logits = output['logits']
        fused_logits = base_logits + gate[..., None, None, None] * satellite_logits
        self._decoder_feature = None
        return {
            **output,
            'base_logits': base_logits,
            'satellite_logits': satellite_logits,
            'satellite_gate': gate,
            'logits': fused_logits,
        }


def r8_satellite_component_loss(
    logits, exact_target, negative_annulus, component_valid,
):
    logits = logits.float()
    exact_target = exact_target.bool()
    negative_annulus = negative_annulus.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    rows = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]]
        negative = logits[batch_index][negative_annulus[batch_index]]
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError('R8 component loss received an empty region.')
        hard_count = min(
            int(negative.numel()),
            max(32, 4 * int(positive.numel())),
        )
        hard_negative = torch.topk(negative, hard_count).values
        positive_bce = F.softplus(-positive).mean()
        negative_bce = F.softplus(hard_negative).mean()
        rank = F.relu(
            1.0 - positive.mean() + hard_negative.mean()
        )
        absolute_floor = F.softplus(
            R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR - positive.mean()
        )
        loss = (
            0.35 * positive_bce + 0.30 * negative_bce
            + 0.20 * rank + 0.15 * absolute_floor
        )
        losses.append(loss)
        rows.append({
            'positive_voxels': int(positive.numel()),
            'negative_voxels': int(negative.numel()),
            'hard_negative_voxels': int(hard_count),
            'positive_logit_mean': float(positive.mean().detach().cpu()),
            'hard_negative_logit_mean': float(
                hard_negative.mean().detach().cpu()
            ),
            'relative_logit_margin': float(
                (positive.mean() - hard_negative.mean()).detach().cpu()
            ),
            'positive_bce': float(positive_bce.detach().cpu()),
            'hard_negative_bce': float(negative_bce.detach().cpu()),
            'rank_loss': float(rank.detach().cpu()),
            'absolute_positive_floor_loss': float(
                absolute_floor.detach().cpu()
            ),
            'total_satellite_loss': float(loss.detach().cpu()),
        })
    if not losses:
        return logits.sum() * 0.0, []
    return torch.stack(losses).mean(), rows


def r8_model_input_batch(sample, device):
    return {
        key: torch.from_numpy(np.asarray(sample[key])[None]).to(device)
        for key in MODEL_INPUT_KEYS_V22
    }


def r8_gradient_norm(module):
    values = [
        parameter.grad.detach().float().norm().cpu().item()
        for parameter in module.parameters()
        if parameter.grad is not None
    ]
    if not values:
        return 0.0, False, True
    finite = bool(np.isfinite(values).all())
    return float(np.linalg.norm(values)), True, finite


def run_r8_dual_resolution_preflight():
    if r8_existing_preflight_complete():
        gate = load_json(R8_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Total parameters: {gate["total_parameters"]:,}')
        print(f'Preflight gate:   {R8_GATE_PATH}')
        print('No source checkpoint or preflight artifact was rewritten.')
        return

    r7_failure_gate = load_json(R7_FAILURE_GATE_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r5_recomputed = dict(
        r7_result['r5_recomputed_under_r7_metric_contract']
    )
    r5_mean_dice = float(r5_recomputed['positive_mean_dice'])
    r5_global_dice = float(r5_recomputed['positive_global_dice'])
    r8_mean_dice_eligibility_floor = r5_mean_dice - 0.005
    r8_global_dice_eligibility_floor = r5_global_dice - 0.010
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_failure_sha = file_sha256(R7_FAILURE_GATE_PATH)
    upstream_checks = {
        'r7_failure_diagnostic_complete': (
            r7_failure_gate.get('diagnostic_complete') is True
        ),
        'r7_failure_diagnostic_passed': (
            r7_failure_gate.get('diagnostic_passed') is True
        ),
        'r8_preflight_allowed_by_diagnostic': (
            r7_failure_gate.get('r8_preflight_implementation_allowed') is True
        ),
        'r8_training_was_locked_by_diagnostic': (
            r7_failure_gate.get('r8_training_authorized') is False
        ),
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r5_recomputed_baseline_is_finite': bool(np.isfinite([
            r5_mean_dice, r5_global_dice,
        ]).all()),
        'r5_checkpoint_hash_matches': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'fold0_fit_count_is_82': len(fold0_fit_df) == 82,
        'fold0_heldout_targets_not_used': len(fold0_heldout_df) == 20,
        'locked_validation_targets_not_used': len(locked_validation_df) == 23,
        'sealed_test_targets_not_used': len(sealed_test_df) == 23,
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError('R8 upstream checks failed: ' + ', '.join(failed))

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT STARTING')
    print('=' * 80)
    print('Warm-start source:                  completed r5 best checkpoint')
    print('R7 checkpoint used for weights:     False')
    print('Target access:                      fold-0 fit probes only')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer / training steps:         0 / 0')

    schedule, schedule_summary, per_epoch, expected_keys, observed_keys = (
        r8_schedule_audit()
    )
    target_probe = r8_native_target_probe(schedule)

    positive_schedule = schedule[
        schedule['sample_type'].astype(str) == 'positive'
    ]
    smoke_candidates = positive_schedule[
        positive_schedule['component_tier'].astype(str).isin(['micro', 'tiny'])
    ]
    smoke_entry = None
    for item in smoke_candidates.to_dict('records'):
        if as_bool(fit_row_by_subject[str(item['subject_id'])][
            'trial8_atlas_enabled'
        ]):
            smoke_entry = item
            break
    if smoke_entry is None:
        raise RuntimeError('R8 found no atlas-enabled micro/tiny smoke case.')
    subject_id = str(smoke_entry['subject_id'])
    row = fit_row_by_subject[subject_id]
    local_origin = pilot_origin_from_center(
        smoke_entry['center_native'], pilot_mask_records[subject_id]['shape']
    )
    dataset = Trial8V22Fold0Dataset()
    dataset.epoch = int(smoke_entry['epoch'])
    dataset.schedule = [smoke_entry]
    sample = dataset[0]
    native_target = r8_native_component_targets(
        row, local_origin, smoke_entry['atomic_component_id']
    )

    if not torch.cuda.is_available():
        raise RuntimeError('R8 preflight requires an L4-class CUDA runtime.')
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < 20.0:
        raise RuntimeError(
            f'R8 preflight expected >=20 GB GPU, observed {gpu_capacity_gb:.2f} GB.'
        )
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'r5_checkpoint_pilot_version_matches': (
            checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'r5_checkpoint_code_revision_matches': (
            checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
        ),
        'r5_checkpoint_parameter_count_matches': int(
            checkpoint.get('parameter_count', -1)
        ) == EXPECTED_V22_PARAMETERS,
        'r5_checkpoint_contains_ema_state': 'ema_model_state' in checkpoint,
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 warm start: ' + ', '.join(failed))
    batch = r8_model_input_batch(sample, device)
    force_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }

    reference = VascuFateNetV22(pilot_schema).to(device).eval()
    reference.load_state_dict(checkpoint['ema_model_state'], strict=True)
    for parameter in reference.parameters():
        parameter.requires_grad_(False)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        reference_output = reference(**batch, **force_flags)
    reference_cpu = {
        key: value.detach().float().cpu()
        for key, value in reference_output.items()
        if torch.is_tensor(value)
    }
    del reference, reference_output
    gc.collect()
    torch.cuda.empty_cache()

    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(checkpoint['ema_model_state'])
    del checkpoint
    model.eval()
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    base_parameters = sum(parameter.numel() for parameter in model.base.parameters())
    satellite_head_parameters = sum(
        parameter.numel() for parameter in model.satellite_head.parameters()
    )
    satellite_gate_parameters = sum(
        parameter.numel() for parameter in model.satellite_gate.parameters()
    )
    new_parameters = satellite_head_parameters + satellite_gate_parameters
    parameter_frame = pd.DataFrame([
        {'component': 'preserved_r5_v2_2_base', 'parameters': base_parameters},
        {'component': 'native_satellite_head', 'parameters': satellite_head_parameters},
        {'component': 'bounded_satellite_gate', 'parameters': satellite_gate_parameters},
        {'component': 'new_r8_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
    ])
    atomic_write_csv(parameter_frame, R8_PARAMETER_PATH)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        neutral_output = model(**batch, **force_flags)
    peak_eval_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    preserved_keys = [
        'logits', 'auxiliary_half_logits', 'auxiliary_quarter_logits',
        'territory_node_logits', 'global_graph_context',
        'global_proposal_logits', 'lesion_presence_logit',
        'log_volume_prediction',
    ]
    neutrality_rows = []
    for key in preserved_keys:
        difference = float(torch.max(torch.abs(
            neutral_output[key].detach().float().cpu() - reference_cpu[key]
        )))
        neutrality_rows.append({
            'output': key,
            'maximum_absolute_difference_from_r5': difference,
            'exactly_preserved_with_zero_gate': difference <= 1e-6,
        })
    neutrality_frame = pd.DataFrame(neutrality_rows)
    atomic_write_csv(neutrality_frame, R8_NEUTRALITY_PATH)

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'base_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_gate': (1, 1),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
    }
    output_rows = []
    for key, expected in expected_shapes.items():
        observed = tuple(neutral_output[key].shape)
        output_rows.append({
            'output': key,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(neutral_output[key]).all().item()),
        })
    output_frame = pd.DataFrame(output_rows)
    atomic_write_csv(output_frame, R8_OUTPUT_PATH)

    zero_gate = float(neutral_output['satellite_gate'].abs().max().cpu())
    satellite_std = float(
        neutral_output['satellite_logits'].float().std().cpu()
    )
    model.activate_satellite_gate_for_gradient_test(0.5)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        activated_output = model(**batch, **force_flags)
    activated_difference = float(torch.max(torch.abs(
        activated_output['logits'].float()
        - activated_output['base_logits'].float()
    )).cpu())
    model.reset_satellite_gate_to_zero()
    del activated_output

    model.train()
    model.zero_grad(set_to_none=True)
    exact_target = torch.from_numpy(
        native_target['native_component_target'][None]
    ).to(device)
    negative_target = torch.from_numpy(
        native_target['native_component_negative_annulus'][None]
    ).to(device)
    component_valid = torch.ones(1, dtype=torch.bool, device=device)
    local_binary = torch.from_numpy(
        np.asarray(sample['target_binary'])[None]
    ).to(device)
    global_target = torch.from_numpy(
        np.asarray(sample['global_target'])[None]
    ).to(device)
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        gradient_output = model(**batch, **force_flags)
        main_loss = (
            0.60 * pilot_soft_dice_loss(
                gradient_output['logits'], local_binary
            ).mean()
            + 0.40 * pilot_focal_bce_loss(
                gradient_output['logits'], local_binary
            ).mean()
        )
        coarse_loss = (
            0.70 * pilot_soft_dice_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
            + 0.30 * pilot_focal_bce_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
        )
        satellite_loss, satellite_loss_rows = r8_satellite_component_loss(
            gradient_output['satellite_logits'], exact_target,
            negative_target, component_valid,
        )
        smoke_loss = main_loss + 0.20 * coarse_loss + 0.10 * satellite_loss
    smoke_loss.backward()
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_groups = {
        'preserved_local_segmentation_head': (
            model.base.local_model.segmentation_head
        ),
        'preserved_global_proposal_stream': model.base.global_stream,
        'native_satellite_head': model.satellite_head,
        'bounded_satellite_gate': model.satellite_gate,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_groups.items():
        norm, observed, finite = r8_gradient_norm(module)
        gradient_rows.append({
            'component': name,
            'gradient_norm': norm,
            'gradient_observed': observed,
            'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_frame = pd.DataFrame(gradient_rows)
    atomic_write_csv(gradient_frame, R8_GRADIENT_PATH)
    loss_frame = pd.DataFrame([{
        'subject_id': subject_id,
        'component_tier': str(smoke_entry['component_tier']),
        'component_id': int(smoke_entry['atomic_component_id']),
        'main_loss': float(main_loss.detach().float().cpu()),
        'coarse_whole_mask_loss': float(coarse_loss.detach().float().cpu()),
        'native_satellite_loss': float(
            satellite_loss.detach().float().cpu()
        ),
        'total_smoke_loss': float(smoke_loss.detach().float().cpu()),
        **satellite_loss_rows[0],
    }])
    atomic_write_csv(loss_frame, R8_LOSS_PATH)
    model.zero_grad(set_to_none=True)

    architecture = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'warm_start': {
            'source': 'completed_r5_best_ema',
            'checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'checkpoint_sha256': r5_sha_before,
            'r7_weights_used': False,
            'zero_gate_output_equivalence_required': True,
        },
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'native_satellite_head': {
            'input': 'native-resolution_r5_decoder_feature_plus_coarse_proposal_plus_arterial_map',
            'input_spatial_shape': list(LOCAL_PATCH_SIZE),
            'hidden_channels': R8_SATELLITE_FEATURE_CHANNELS,
            'output': 'native_resolution_satellite_logits',
            'direct_graph_to_logit_residual': False,
        },
        'fusion': {
            'type': 'zero_initialized_bounded_scalar_residual',
            'bound': R8_SATELLITE_GATE_BOUND,
            'gate_context': 'registration_confidence_plus_lesion_presence',
            'initial_final_output': 'exact_r5_logits',
        },
        'coarse_global_role': (
            'whole_brain_burden_and_broad_localization_only; '
            'no_r7_coarse_component_contrastive_loss'
        ),
        'clinical_and_arterial_role': (
            'preserved_bounded_v2_2_context; no treatment/outcome leakage'
        ),
    }
    training_policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'experiment': 'R8_fold0_pilot_only_after_separate_training_preflight',
        'component_cycle': {
            'positive_patches_per_epoch': R8_COMPONENT_POSITIVE_PATCHES,
            'tier_quotas': R8_TIER_QUOTAS,
            'rule': 'within_each_tier_every_component_once_before_repeat',
            'schedule_epochs_audited': R8_SCHEDULE_EPOCHS,
        },
        'optimization_stages': [
            {
                'epochs': '1-4',
                'trainable': ['native_satellite_head'],
                'frozen': ['r5_base', 'satellite_gate'],
                'purpose': 'learn_absolute_native_component_evidence_before_fusion',
            },
            {
                'epochs': '5-12',
                'trainable': ['native_satellite_head', 'satellite_gate'],
                'frozen': ['r5_base'],
                'purpose': 'learn_bounded_residual_without_erasing_r5_behavior',
            },
            {
                'epochs': '13+',
                'trainable': ['all'],
                'learning_rate_policy': (
                    'r5_base_2e-5_satellite_2e-4_gate_1e-4'
                ),
                'purpose': 'low_rate_joint_calibration',
            },
        ],
        'loss_contract': {
            'main_segmentation': 'preserved_r5_dice_plus_focal',
            'coarse_proposal': 'whole_mask_only',
            'native_satellite': (
                'per_component_positive_bce_plus_hard_negative_bce_plus_'
                'ranking_plus_absolute_positive_logit_floor'
            ),
            'coarse_component_contrastive': False,
            'volume_case_weights': False,
            'topk_ohem': False,
        },
        'checkpoint_eligibility': {
            'r5_recomputed_mean_dice': r5_mean_dice,
            'r5_recomputed_global_dice': r5_global_dice,
            'mean_dice_minimum': r8_mean_dice_eligibility_floor,
            'global_dice_minimum': r8_global_dice_eligibility_floor,
            'small_component_improvement_required': True,
            'deployment_metrics_only': True,
            'oracle_volume_budget_used_for_selection': False,
            'selection_order': [
                'eligibility_guards', 'mean_dice', 'small_case_dice',
                'small_component_recall', 'lesion_f1',
            ],
        },
        'fold0_heldout_targets_for_training': False,
        'locked_validation_targets_for_training': False,
        'test_targets_for_training': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(architecture, R8_ARCHITECTURE_PATH)
    atomic_write_json(training_policy, R8_TRAINING_POLICY_PATH)

    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    preflight_checks = {
        **upstream_checks,
        **checkpoint_checks,
        'tier_quotas_sum_to_112': sum(R8_TIER_QUOTAS.values()) == 112,
        'schedule_has_40_epochs': schedule['epoch'].nunique() == 40,
        'every_epoch_has_168_samples': bool(
            (schedule.groupby('epoch').size() == 168).all()
        ),
        'every_epoch_has_112_positive_samples': bool(
            (schedule[schedule['sample_type'] == 'positive']
             .groupby('epoch').size() == 112).all()
        ),
        'all_eligible_components_are_covered': observed_keys == expected_keys,
        'no_eligible_component_is_never_selected': bool(
            (schedule_summary['never_selected_components'] == 0).all()
        ),
        'component_selection_counts_differ_by_at_most_one': bool(
            (schedule_summary['maximum_minus_minimum_selections'] <= 1).all()
        ),
        'complete_tier_cycles_have_no_duplicates': bool(
            schedule_summary['complete_cycles_have_no_duplicates'].all()
        ),
        'component_cycle_gini_below_0p10': bool(
            (schedule_summary['selection_gini'] < 0.10).all()
        ),
        'native_target_probes_cover_four_tiers': (
            target_probe['component_tier'].nunique() == 4
        ),
        'all_native_target_probes_are_nonempty': bool(
            (target_probe['component_patch_voxels'] > 0).all()
        ),
        'all_native_annuli_are_nonempty': bool(
            (target_probe['negative_annulus_voxels'] > 0).all()
        ),
        'all_native_targets_and_annuli_are_disjoint': bool(
            target_probe['exact_and_annulus_disjoint'].all()
        ),
        'all_native_target_probes_retain_at_least_95_percent': bool(
            (target_probe['component_retained_fraction'] >= 0.95).all()
        ),
        'preserved_base_parameter_count_matches_r5': (
            base_parameters == EXPECTED_V22_PARAMETERS
        ),
        'new_parameter_count_within_limit': (
            0 < new_parameters <= R8_MAXIMUM_NEW_PARAMETERS
        ),
        'all_zero_gate_outputs_match_r5': bool(
            neutrality_frame['exactly_preserved_with_zero_gate'].all()
        ),
        'satellite_gate_is_exactly_zero_initially': zero_gate == 0.0,
        'satellite_logits_are_nonconstant': satellite_std > 1e-6,
        'activated_satellite_gate_changes_final_logits': (
            activated_difference > 1e-7
        ),
        'all_output_shapes_match': bool(output_frame['shape_matches'].all()),
        'all_outputs_are_finite': bool(output_frame['finite'].all()),
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss).item()),
        'satellite_loss_contains_absolute_positive_floor': (
            float(loss_frame['absolute_positive_floor_loss'].iloc[0]) > 0.0
        ),
        **gradient_checks,
        'peak_eval_memory_below_20gb': peak_eval_memory_gb < 20.0,
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'r7_checkpoint_not_loaded_for_weights': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in preflight_checks.items()
    ])
    atomic_write_csv(checks_frame, R8_CHECKS_PATH)
    preflight_passed = all(preflight_checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R8_OUTPUT_ARTIFACTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_used_for_weights': False,
        'r7_failure_diagnostic_path': str(R7_FAILURE_GATE_PATH),
        'r7_failure_diagnostic_sha256': r7_failure_sha,
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'gpu': gpu_name,
        'gpu_capacity_gb': gpu_capacity_gb,
        'peak_eval_memory_gb': peak_eval_memory_gb,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'schedule_components': len(expected_keys),
        'minimum_unique_positive_subjects_per_epoch': int(
            per_epoch['unique_subjects'].min()
        ),
        'native_target_probes': len(target_probe),
        'optimizer_steps': 0,
        'training_performed': False,
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'r8_pilot_implementation_allowed': bool(preflight_passed),
        'r8_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'checks': preflight_checks,
        'artifact_sha256': artifact_hashes,
        'architecture_path': str(R8_ARCHITECTURE_PATH),
        'training_policy_path': str(R8_TRAINING_POLICY_PATH),
    }
    atomic_write_json(gate, R8_GATE_PATH)
    del model, neutral_output, gradient_output, batch
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in preflight_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR8 component-cycle summary:')
    display(schedule_summary)
    print('\nR8 parameter count:')
    display(parameter_frame)
    print('\nR8 output shapes:')
    display(output_frame)
    print('\nR8 gradient smoke:')
    display(gradient_frame)
    print('\nR8 preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Warm-start checkpoint:              r5 epoch {r5_result["best_epoch"]}')
    print('R7 weights used:                    False')
    print(f'Preserved base parameters:          {base_parameters:,}')
    print(f'New satellite parameters:           {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Eligible components cycled:         {len(expected_keys)}')
    print(f'Minimum unique subjects per epoch:  {int(per_epoch["unique_subjects"].min())}')
    print(f'Peak gradient memory GB:            {peak_gradient_memory_gb:.3f}')
    print('Coarse component contrastive loss:  disabled')
    print('Native absolute calibration loss:   enabled')
    print('Initial final logits equal r5:      True')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Training performed:                 False')
    print('R8 pilot implementation allowed:    True')
    print('R8 pilot training authorized:       False')
    print('Full five-fold training authorized: False')
    print(f'Architecture:                       {R8_ARCHITECTURE_PATH}')
    print(f'Training policy:                    {R8_TRAINING_POLICY_PATH}')
    print(f'Preflight gate:                     {R8_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 R8 pilot loader/loss/training '
        'preflight from this frozen contract; do not train yet.'
    )


run_r8_dual_resolution_preflight()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = (
    'r7_delayed_local_contrastive_volume_bounded_r1'
)
PILOT_VERSION = 'v2_2_fold0_local_contrastive_proposal_r7'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

R7_PREFLIGHT_VERSION = 'v1_delayed_small_component_local_contrastive'
R7_PREFLIGHT_CODE_REVISION = (
    'r1_precision_bounded_diffuse_shortcut_guard'
)
R7_COMPONENT_LOSS_MAX_COEFFICIENT = 0.10
R7_COMPONENT_LOSS_START_EPOCH = 8
R7_COMPONENT_LOSS_FULL_EPOCH = 12
R7_POSITIVE_WEIGHT = 0.40
R7_HARD_NEGATIVE_WEIGHT = 0.40
R7_RANK_WEIGHT = 0.20
R7_INNER_RADIUS_MM = 4.0
R7_OUTER_RADIUS_MM = 12.0
R7_HARD_NEGATIVE_RATIO = 4
R7_HARD_NEGATIVE_MINIMUM = 16
R7_LOGIT_MARGIN = 1.0
R7_VOLUME_BUDGET_MULTIPLIER = 3.0
R7_SMALL_COMPONENT_TIERS = set(COMPONENT_TIERS[:4])
R5_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
R5_PILOT_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
R7_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip((0.8, 0.8, 2.0), GLOBAL_DOWNSAMPLE)
)

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'
R7_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r7_local_contrastive_component_proposal' / R7_PREFLIGHT_VERSION
)
R7_PREFLIGHT_GATE_PATH = (
    R7_PREFLIGHT_ROOT / 'r7_local_contrastive_preflight_gate.json'
)
R7_POLICY_PATH = R7_PREFLIGHT_ROOT / 'r7_local_contrastive_policy.json'
R5_PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / R5_PILOT_VERSION
R5_RESULT_GATE_PATH = R5_PILOT_ROOT / 'pilot_result_gate.json'
R5_BEST_CHECKPOINT_PATH = R5_PILOT_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'
R7_R5_BASELINE_DIAGNOSTIC_PATH = (
    PILOT_DIAGNOSTIC_ROOT / 'r5_volume_bounded_baseline.json'
)
R7_FAILURE_DIAGNOSTIC_VERSION = 'v1_r5_r7_failure_mechanism_audit'
R7_FAILURE_DIAGNOSTIC_CODE_REVISION = (
    'r1_exposure_resolution_margin_volume_branch_checkpoint'
)
R7_FAILURE_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'model_diagnostics'
    / MODEL_VERSION / R7_FAILURE_DIAGNOSTIC_VERSION
)
R7_FAILURE_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
R7_FAILURE_GATE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_diagnostic_gate.json'
)
R7_FAILURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_summary.json'
)



PILOT_TARGET_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'fold0_target_access_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_schedule_audit.csv'
)
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH, R7_PREFLIGHT_GATE_PATH, R7_POLICY_PATH,
    R5_RESULT_GATE_PATH, R5_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
r7_preflight_gate = load_json(R7_PREFLIGHT_GATE_PATH)
r7_policy = load_json(R7_POLICY_PATH)
r5_result_gate = load_json(R5_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r7_component_preflight_passed': (
        r7_preflight_gate.get('preflight_passed') is True
    ),
    'r7_component_preflight_all_checks_true': (
        len(r7_preflight_gate.get('checks', {})) > 0
        and all(bool(value) for value in r7_preflight_gate['checks'].values())
    ),
    'r7_preflight_version_matches': (
        r7_preflight_gate.get('preflight_version') == R7_PREFLIGHT_VERSION
    ),
    'r7_preflight_revision_matches': (
        r7_preflight_gate.get('code_revision')
        == R7_PREFLIGHT_CODE_REVISION
    ),
    'r7_implementation_allowed': (
        r7_preflight_gate.get('r7_pilot_implementation_allowed') is True
    ),
    'r7_training_was_locked_before_this_block': (
        r7_preflight_gate.get('r7_pilot_training_authorized') is False
    ),
    'r7_architecture_parameter_count_unchanged': (
        int(r7_preflight_gate.get('architecture_parameters', -1))
        == EXPECTED_V22_PARAMETERS
        and int(r7_preflight_gate.get('new_trainable_parameters', -1)) == 0
    ),
    'r7_preflight_target_scope_is_fit_only': (
        r7_preflight_gate.get('target_arrays_read_splits') == ['fold0_fit']
        and r7_preflight_gate.get('fold0_heldout_targets_read') is False
        and r7_preflight_gate.get('locked_validation_targets_read') is False
        and r7_preflight_gate.get('test_targets_read') is False
    ),
    'r7_policy_path_matches_gate': (
        str(r7_preflight_gate.get('policy_path')) == str(R7_POLICY_PATH)
    ),
    'r7_policy_local_contrastive_contract_matches': (
        r7_policy.get('selected_component_objective', {}).get(
            'eligible_tiers'
        ) == sorted(R7_SMALL_COMPONENT_TIERS)
        and int(r7_policy.get('selected_component_objective', {}).get(
            'start_epoch', -1
        )) == R7_COMPONENT_LOSS_START_EPOCH
        and int(r7_policy.get('selected_component_objective', {}).get(
            'full_coefficient_epoch', -1
        )) == R7_COMPONENT_LOSS_FULL_EPOCH
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('maximum_coefficient', float('nan'))),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('inner_radius_mm', float('nan'))),
            R7_INNER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('outer_radius_mm', float('nan'))),
            R7_OUTER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('positive_weight', float('nan'))),
            R7_POSITIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_weight', float('nan'))),
            R7_HARD_NEGATIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('rank_weight', float('nan'))),
            R7_RANK_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_ratio', -1)) == R7_HARD_NEGATIVE_RATIO
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_minimum', -1)) == R7_HARD_NEGATIVE_MINIMUM
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('logit_margin', float('nan'))),
            R7_LOGIT_MARGIN, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get('experiment_seed', -1)) == PILOT_SEED
    ),
    'r7_policy_requires_volume_bounded_diagnostics': (
        r7_policy.get('proposal_diagnostics', {}).get(
            'raw_distance_metrics_are_not_success_gates'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_proposal_precision'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_predicted_to_target_volume_ratio'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_voxel_average_precision'
        ) is True
        and math.isclose(float(r7_policy.get(
            'proposal_diagnostics', {}
        ).get('volume_budget_multiplier', float('nan'))),
            R7_VOLUME_BUDGET_MULTIPLIER, rel_tol=0.0, abs_tol=1e-12)
    ),
    'r7_policy_preserves_whole_mask_loss': (
        math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('soft_dice_weight', float('nan'))),
            0.65, rel_tol=0.0, abs_tol=1e-12,
        )
        and math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('focal_bce_weight', float('nan'))),
            0.35, rel_tol=0.0, abs_tol=1e-12,
        )
    ),
    'r7_checkpoint_policy_is_decoupled': (
        r7_policy.get('checkpoint_policy', {}).get('save_checkpoint_on')
        == 'every strict numeric score improvement'
        and r7_policy.get('checkpoint_policy', {}).get('minimum_delta_use')
        == 'early-stopping patience reset only'
        and r7_policy.get('checkpoint_policy', {}).get(
            'minimum_delta_must_not_block_checkpoint_save'
        ) is True
    ),
    'r5_read_only_baseline_gate_matches': (
        r5_result_gate.get('pilot_version') == R5_PILOT_VERSION
        and r5_result_gate.get('code_revision') == R5_PILOT_CODE_REVISION
        and r5_result_gate.get('pilot_complete') is True
        and str(r5_result_gate.get('best_checkpoint_path'))
        == str(R5_BEST_CHECKPOINT_PATH)
        and r5_result_gate.get('best_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def r7_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


R7_INNER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_INNER_RADIUS_MM
)
R7_OUTER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_OUTER_RADIUS_MM
)


def r7_component_coefficient(epoch):
    epoch = int(epoch)
    if epoch < R7_COMPONENT_LOSS_START_EPOCH:
        return 0.0
    progress = (
        (epoch - R7_COMPONENT_LOSS_START_EPOCH + 1)
        / (R7_COMPONENT_LOSS_FULL_EPOCH - R7_COMPONENT_LOSS_START_EPOCH + 1)
    )
    return float(
        R7_COMPONENT_LOSS_MAX_COEFFICIENT * np.clip(progress, 0.0, 1.0)
    )


R7_COMPONENT_COEFFICIENT_SCHEDULE = {
    epoch: r7_component_coefficient(epoch)
    for epoch in range(1, MAX_EPOCHS + 1)
}
R7_EXPECTED_RAMP = {
    8: 0.02, 9: 0.04, 10: 0.06, 11: 0.08, 12: 0.10,
}
if not all(
    R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch] == 0.0
    for epoch in range(1, R7_COMPONENT_LOSS_START_EPOCH)
):
    raise RuntimeError('R7 local-contrastive coefficient is nonzero before epoch 8.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch], expected,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch, expected in R7_EXPECTED_RAMP.items()
):
    raise RuntimeError('R7 local-contrastive coefficient ramp changed.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch],
        R7_COMPONENT_LOSS_MAX_COEFFICIENT,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch in range(R7_COMPONENT_LOSS_FULL_EPOCH, MAX_EPOCHS + 1)
):
    raise RuntimeError('R7 local-contrastive coefficient does not remain capped.')


def r7_component_regions(exact_target, full_target, valid_domain):
    exact = np.asarray(exact_target, dtype=bool)
    full = np.asarray(full_target) > 0
    domain = np.asarray(valid_domain, dtype=bool)
    inner = ndimage.binary_dilation(exact, structure=R7_INNER_STRUCTURE)
    outer = ndimage.binary_dilation(exact, structure=R7_OUTER_STRUCTURE)
    negative = outer & ~inner & domain & ~full
    return inner, outer, negative


def pilot_extract_targets(
    row, local_origin, sample_type, atomic_component_id, component_tier
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary > 0
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(binary, dtype=np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if start_a != start_b or start_a != start_c:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_domain = F.max_pool3d(
        torch.from_numpy(domain_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0

    selected_sample = str(sample_type) == 'positive'
    selected_valid = (
        selected_sample and str(component_tier) in R7_SMALL_COMPONENT_TIERS
    )
    selected_component = np.zeros_like(component_support, dtype=bool)
    if selected_sample:
        labels, count = ndimage.label(
            component_support, structure=np.ones((3, 3, 3), dtype=bool)
        )
        expected_count = len(component_records.get(subject_id, []))
        if count != expected_count:
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {expected_count}.'
            )
        component_id = int(atomic_component_id)
        if component_id < 1 or component_id > count:
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is outside '
                f'the frozen 1..{count} range.'
            )
        selected_component = labels == component_id
        if not bool(selected_component.any()):
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is empty.'
            )

    selected_canvas, selected_start = symmetric_pad(
        selected_component.astype(np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if selected_start != start_a:
        raise RuntimeError(
            f'{subject_id}: selected-component padding start disagrees.'
        )
    selected_global = F.max_pool3d(
        torch.from_numpy(selected_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0
    selected_negative = np.zeros_like(selected_global, dtype=bool)
    if selected_valid:
        _, _, selected_negative = r7_component_regions(
            selected_global, global_target[0], global_domain
        )
        if not bool(selected_negative.any()):
            raise RuntimeError(
                f'{subject_id}: selected component has no lesion-free '
                '4-12 mm negative annulus.'
            )
    if selected_sample:
        frozen = [
            item for item in component_records[subject_id]
            if int(item['atomic_component_id']) == int(atomic_component_id)
        ]
        if len(frozen) != 1:
            raise RuntimeError(
                f'{subject_id}: selected component does not map uniquely to '
                'the frozen component audit.'
            )
        expected_global_voxels = int(frozen[0]['global_proposal_voxels'])
        if int(selected_global.sum()) != expected_global_voxels:
            raise RuntimeError(
                f'{subject_id}: selected component has '
                f'{int(selected_global.sum())} pooled voxels; expected '
                f'{expected_global_voxels}.'
            )
        if bool((selected_negative & selected_global).any()):
            raise RuntimeError(
                f'{subject_id}: negative annulus overlaps exact component.'
            )
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'selected_component_target': selected_global[None].astype(bool),
        'selected_component_negative_annulus': (
            selected_negative[None].astype(bool)
        ),
        'selected_component_valid': np.asarray(
            selected_valid, dtype=bool
        ),
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'selected_component_target': targets[
                'selected_component_target'
            ],
            'selected_component_negative_annulus': targets[
                'selected_component_negative_annulus'
            ],
            'selected_component_valid': targets[
                'selected_component_valid'
            ],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def r7_local_contrastive_loss(
    logits, exact_component_target, negative_annulus_target, component_valid,
):
    exact_target = exact_component_target.bool()
    negative_target = negative_annulus_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    positive_losses = []
    negative_losses = []
    rank_losses = []
    positive_logits = []
    hard_negative_logits = []
    selected_negative_counts = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]].float()
        negative = logits[batch_index][negative_target[batch_index]].float()
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError(
                'A valid r7 component must have positive voxels and a '
                'lesion-free negative annulus.'
            )
        hard_count = min(
            int(negative.numel()),
            max(
                R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * int(positive.numel()),
            ),
        )
        hard_negative = torch.topk(
            negative, k=hard_count, largest=True
        ).values
        positive_loss = F.softplus(-positive).mean()
        negative_loss = F.softplus(hard_negative).mean()
        rank_loss = F.softplus(
            R7_LOGIT_MARGIN - positive.mean() + hard_negative.mean()
        )
        contrastive_loss = (
            R7_POSITIVE_WEIGHT * positive_loss
            + R7_HARD_NEGATIVE_WEIGHT * negative_loss
            + R7_RANK_WEIGHT * rank_loss
        )
        losses.append(contrastive_loss)
        positive_losses.append(positive_loss)
        negative_losses.append(negative_loss)
        rank_losses.append(rank_loss)
        positive_logits.append(positive.mean())
        hard_negative_logits.append(hard_negative.mean())
        selected_negative_counts.append(hard_count)

    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'local_contrastive_loss': zero,
            'component_positive_loss': zero,
            'component_hard_negative_loss': zero,
            'component_rank_loss': zero,
            'mean_component_positive_logit': zero.detach(),
            'mean_component_hard_negative_logit': zero.detach(),
            'selected_hard_negative_voxels': zero.detach(),
            'selected_components_in_batch': zero.detach(),
        }
    return torch.stack(losses).mean(), {
        'local_contrastive_loss': torch.stack(losses).mean(),
        'component_positive_loss': torch.stack(positive_losses).mean(),
        'component_hard_negative_loss': torch.stack(negative_losses).mean(),
        'component_rank_loss': torch.stack(rank_losses).mean(),
        'mean_component_positive_logit': torch.stack(positive_logits).mean(),
        'mean_component_hard_negative_logit': torch.stack(
            hard_negative_logits
        ).mean(),
        'selected_hard_negative_voxels': torch.as_tensor(
            float(sum(selected_negative_counts)), device=logits.device
        ),
        'selected_components_in_batch': torch.as_tensor(
            float(len(losses)), device=logits.device
        ),
    }


def r7_local_contrastive_proposal_loss(
    logits, full_target, exact_component_target,
    negative_annulus_target, component_valid, epoch,
):
    whole_dice = pilot_soft_dice_loss(logits, full_target).mean()
    whole_bce = pilot_focal_bce_loss(logits, full_target).mean()
    whole_loss = 0.65 * whole_dice + 0.35 * whole_bce
    contrastive_loss, contrastive_parts = r7_local_contrastive_loss(
        logits, exact_component_target, negative_annulus_target,
        component_valid,
    )
    coefficient = r7_component_coefficient(epoch)
    proposal_loss = whole_loss + coefficient * contrastive_loss
    return proposal_loss, {
        'proposal_whole_mask_loss': whole_loss,
        'proposal_dice_loss': whole_dice,
        'proposal_focal_bce_loss': whole_bce,
        **contrastive_parts,
        'component_loss_coefficient': torch.as_tensor(
            coefficient, device=logits.device, dtype=torch.float32
        ),
    }


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(
    model, cpu_batch, device, epoch, batch_index,
    dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    if dropout_flags_override is not None:
        required_flags = {
            'force_atlas_dropout', 'force_vascular_dropout',
            'force_occlusion_dropout', 'force_clinical_dropout',
            'force_global_dropout', 'force_arterial_dropout',
        }
        if set(dropout_flags_override) != required_flags:
            raise RuntimeError(
                'Dropout override keys do not match the frozen branch set.'
            )
        flags = {
            name: bool(dropout_flags_override[name])
            for name in sorted(required_flags)
        }
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_loss, proposal_parts = r7_local_contrastive_proposal_loss(
        output['global_proposal_logits'], global_target,
        batch['selected_component_target'],
        batch['selected_component_negative_annulus'],
        batch['selected_component_valid'], epoch,
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        **proposal_parts,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'arterial_valid_nodes': arterial_valid.sum().detach().float(),
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'arterial_dropped': torch.as_tensor(float(flags['force_arterial_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_volume_bounded_prediction(probability, target_voxels):
    probability = np.asarray(probability, dtype=np.float32)
    budget = min(
        probability.size,
        max(1, int(math.ceil(
            R7_VOLUME_BUDGET_MULTIPLIER * int(target_voxels)
        ))),
    )
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape), budget


def pilot_histogram_average_precision(positive_histogram, negative_histogram):
    positive = np.asarray(positive_histogram, dtype=np.float64)[::-1]
    negative = np.asarray(negative_histogram, dtype=np.float64)[::-1]
    total_positive = float(positive.sum())
    if total_positive <= 0:
        return float('nan')
    cumulative_positive = np.cumsum(positive)
    cumulative_total = np.cumsum(positive + negative)
    precision = cumulative_positive / np.maximum(cumulative_total, 1.0)
    return float((precision * positive).sum() / total_positive)


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    voxel_precision = total_intersection / max(total_prediction, 1)
    predicted_to_target_volume_ratio = (
        total_prediction / max(total_target, 1)
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'voxel_precision': float(voxel_precision),
        'predicted_to_target_volume_ratio': float(
            predicted_to_target_volume_ratio
        ),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['volume_calibration_score'] = np.exp(-np.abs(
        np.log(np.maximum(
            proposal_threshold_df['predicted_to_target_volume_ratio'], 1e-6
        ))
    ))
    proposal_threshold_df['proposal_selection_score'] = (
        0.35 * proposal_threshold_df['positive_global_dice']
        + 0.15 * proposal_threshold_df['lesion_f1']
        + 0.20 * proposal_threshold_df['voxel_precision']
        + 0.15 * proposal_threshold_df['component_recall']
        + 0.15 * proposal_threshold_df['volume_calibration_score']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        [
            'proposal_selection_score', 'positive_global_dice',
            'voxel_precision', 'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    proposal_ap_positive_histogram = np.zeros(256, dtype=np.int64)
    proposal_ap_negative_histogram = np.zeros(256, dtype=np.int64)
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        proposal_intersection = int((proposal_prediction & target).sum())
        proposal_predicted_voxels = int(proposal_prediction.sum())
        proposal_distance = None
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        budgeted_proposal_prediction = None
        budgeted_proposal_distance = None
        proposal_volume_budget_voxels = 0
        if hard_eligible and target_voxels > 0:
            (
                budgeted_proposal_prediction,
                proposal_volume_budget_voxels,
            ) = pilot_volume_bounded_prediction(proposal, target_voxels)
            budgeted_proposal_distance = ndimage.distance_transform_edt(
                ~budgeted_proposal_prediction, sampling=case['spacing']
            )
            proposal_ap_positive_histogram += np.histogram(
                proposal[target], bins=256, range=(0.0, 1.0)
            )[0]
            proposal_ap_negative_histogram += np.histogram(
                proposal[~target], bins=256, range=(0.0, 1.0)
            )[0]
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
            'proposal_predicted_voxels': proposal_predicted_voxels,
            'proposal_intersection_voxels': proposal_intersection,
            'proposal_voxel_precision': (
                proposal_intersection / max(proposal_predicted_voxels, 1)
            ),
            'proposal_voxel_recall': (
                proposal_intersection / max(target_voxels, 1)
            ),
            'proposal_predicted_to_target_volume_ratio': (
                proposal_predicted_voxels / max(target_voxels, 1)
            ),
            'proposal_volume_budget_voxels': proposal_volume_budget_voxels,
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            budgeted_proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    budgeted_proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                budgeted_proposal_max_iou = float(
                    budgeted_proposal_iou[component_id][
                        'maximum_component_iou'
                    ]
                )
                proposal_distance_mm = float('nan')
                if proposal_distance is not None:
                    proposal_distance_mm = float(
                        proposal_distance[component_slice][component_local].min(
                            initial=np.inf
                        )
                    )
                budgeted_proposal_distance_mm = float(
                    budgeted_proposal_distance[component_slice][
                        component_local
                    ].min(initial=np.inf)
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                budgeted_proposal_covered = (
                    budgeted_proposal_max_iou >= 0.05
                )
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += int(budgeted_proposal_covered)
                tier_counts[tier][3] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'proposal_minimum_distance_mm': proposal_distance_mm,
                    'budgeted_proposal_maximum_component_iou': (
                        budgeted_proposal_max_iou
                    ),
                    'budgeted_proposal_minimum_distance_mm': (
                        budgeted_proposal_distance_mm
                    ),
                    'proposal_within_2mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 2.0
                    ),
                    'proposal_within_4mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 4.0
                    ),
                    'proposal_within_8mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_within_2mm': bool(
                        budgeted_proposal_distance_mm <= 2.0
                    ),
                    'budgeted_proposal_within_4mm': bool(
                        budgeted_proposal_distance_mm <= 4.0
                    ),
                    'budgeted_proposal_within_8mm': bool(
                        budgeted_proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_covered': bool(
                        budgeted_proposal_covered
                    ),
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, budgeted_proposal_tp, total = tier_counts[tier]
        tier_components = [
            item for item in component_rows
            if item['component_tier'] == tier
        ]
        finite_distances = [
            float(item['proposal_minimum_distance_mm'])
            for item in tier_components
            if np.isfinite(item['proposal_minimum_distance_mm'])
        ]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
            'budgeted_proposal_component_recall': (
                budgeted_proposal_tp / max(total, 1)
            ),
            'proposal_within_2mm_fraction': float(np.mean([
                item['proposal_within_2mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_4mm_fraction': float(np.mean([
                item['proposal_within_4mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_8mm_fraction': float(np.mean([
                item['proposal_within_8mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_2mm_fraction': float(np.mean([
                item['budgeted_proposal_within_2mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_4mm_fraction': float(np.mean([
                item['budgeted_proposal_within_4mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_8mm_fraction': float(np.mean([
                item['budgeted_proposal_within_8mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'median_finite_proposal_distance_mm': float(
                np.median(finite_distances)
            ) if finite_distances else float('nan'),
            'mean_proposal_maximum_probability': float(np.mean([
                item['proposal_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'mean_final_maximum_probability': float(np.mean([
                item['final_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'representation_failure_fraction': float(np.mean([
                item['representation_failure_below_0p10']
                for item in tier_components
            ])) if tier_components else 0.0,
        })
    small_component_rows = [
        item for item in component_rows
        if item['component_tier'] in set(COMPONENT_TIERS[:4])
    ]
    proposal_voxel_average_precision = pilot_histogram_average_precision(
        proposal_ap_positive_histogram, proposal_ap_negative_histogram
    )
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'proposal_voxel_precision': float(
            proposal_selected['voxel_precision']
        ),
        'proposal_predicted_to_target_volume_ratio': float(
            proposal_selected['predicted_to_target_volume_ratio']
        ),
        'proposal_volume_calibration_score': float(
            proposal_selected['volume_calibration_score']
        ),
        'proposal_voxel_average_precision_256bin': float(
            proposal_voxel_average_precision
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'small_components_audited': len(small_component_rows),
        'small_component_proposal_within_2mm_fraction': float(np.mean([
            item['proposal_within_2mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_4mm_fraction': float(np.mean([
            item['proposal_within_4mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_8mm_fraction': float(np.mean([
            item['proposal_within_8mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_budgeted_proposal_within_2mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_2mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_4mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_4mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_8mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_8mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered'] for item in component_rows
        ])) if component_rows else 0.0,
        'small_budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_representation_failure_fraction': float(np.mean([
            item['representation_failure_below_0p10']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    required_finite_summary_metrics = [
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'proposal_global_dice', 'proposal_component_recall',
        'proposal_voxel_precision',
        'proposal_predicted_to_target_volume_ratio',
        'proposal_voxel_average_precision_256bin',
        'budgeted_proposal_component_recall',
        'small_budgeted_proposal_component_recall',
        'small_component_budgeted_proposal_within_2mm_fraction',
        'small_component_budgeted_proposal_within_4mm_fraction',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'small_component_representation_failure_fraction',
        'pilot_checkpoint_score',
    ]
    nonfinite_summary_metrics = [
        name for name in required_finite_summary_metrics
        if not np.isfinite(float(summary[name]))
    ]
    if nonfinite_summary_metrics:
        raise RuntimeError(
            'Validation produced non-finite required metrics: '
            + ', '.join(nonfinite_summary_metrics)
        )
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
            and completed.get('best_checkpoint_sha256')
            == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R7 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_index = None
    smoke_subject_id = None
    for candidate_index, entry in enumerate(dataset.schedule):
        if entry['sample_type'] != 'positive':
            continue
        candidate_subject = str(entry['subject_id'])
        if not as_bool(
            fit_row_by_subject[candidate_subject]['trial8_atlas_enabled']
        ):
            continue
        candidate_sample = dataset[candidate_index]
        candidate_checks = {
            'atlas_enabled': bool(np.asarray(
                candidate_sample['atlas_enabled']
            ).reshape(-1)[0]),
            'territory_nodes_valid': bool(np.asarray(
                candidate_sample['territory_valid']
            ).astype(bool).any()),
            'vessel_nodes_valid': bool(np.asarray(
                candidate_sample['vessel_valid']
            ).astype(bool).any()),
            'selected_component_valid': bool(np.asarray(
                candidate_sample['selected_component_valid']
            ).reshape(-1)[0]),
        }
        if all(candidate_checks.values()):
            smoke_index = int(candidate_index)
            smoke_subject_id = candidate_subject
            break
    if smoke_index is None:
        raise RuntimeError(
            'No positive fold-0-fit sample has simultaneous atlas, territory, '
            'vessel, and selected-component support for the all-branch smoke '
            'test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    smoke_dropout_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device,
            epoch=R7_COMPONENT_LOSS_FULL_EPOCH, batch_index=0,
            dropout_flags_override=smoke_dropout_flags,
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    r7_smoke_checks = {
        'positive_sample_selected': bool(
            smoke_batch['selected_component_valid'].bool().all().item()
        ),
        'atlas_enabled_for_smoke': bool(
            smoke_batch['atlas_enabled'].bool().all().item()
        ),
        'territory_support_for_smoke': bool(
            smoke_batch['territory_valid'].bool().any().item()
        ),
        'vessel_support_for_smoke': bool(
            smoke_batch['vessel_valid'].bool().any().item()
        ),
        'projected_arterial_nodes_valid_for_smoke': float(
            smoke_parts['arterial_valid_nodes'].detach().float().cpu()
        ) > 0.0,
        'arterial_loss_finite_for_smoke': bool(torch.isfinite(
            smoke_parts['arterial_loss']
        ).item()),
        'all_branch_dropouts_disabled_for_smoke': not any(
            smoke_dropout_flags.values()
        ),
        'exact_component_target_nonempty': bool(
            smoke_batch['selected_component_target'].bool().any().item()
        ),
        'negative_annulus_nonempty': bool(
            smoke_batch['selected_component_negative_annulus']
            .bool().any().item()
        ),
        'negative_annulus_disjoint_from_exact_component': not bool(torch.any(
            smoke_batch['selected_component_target'].bool()
            & smoke_batch['selected_component_negative_annulus'].bool()
        ).item()),
        'local_contrastive_loss_positive': float(
            smoke_parts['local_contrastive_loss']
            .detach().float().cpu()
        ) > 0.0,
        'smoke_uses_maximum_contrastive_coefficient': math.isclose(
            float(smoke_parts['component_loss_coefficient']
                  .detach().float().cpu()),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT,
            rel_tol=0.0, abs_tol=1e-7,
        ),
        'whole_mask_proposal_loss_finite': bool(torch.isfinite(
            smoke_parts['proposal_whole_mask_loss']
        ).item()),
    }
    if not all(r7_smoke_checks.values()):
        failed = [
            name for name, passed in r7_smoke_checks.items() if not passed
        ]
        raise RuntimeError(
            'R7 local-contrastive smoke checks failed: ' + ', '.join(failed)
        )
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    if R7_R5_BASELINE_DIAGNOSTIC_PATH.exists():
        r5_baseline_diagnostic = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
        if (
            r5_baseline_diagnostic.get('r5_checkpoint_sha256')
            != r5_checkpoint_sha256
            or r5_baseline_diagnostic.get('metric_contract_revision')
            != PILOT_CODE_REVISION
        ):
            raise RuntimeError(
                'Existing r5 volume-bounded baseline is incompatible with '
                'the current r7 contract.'
            )
        r5_recomputed_summary = dict(
            r5_baseline_diagnostic['validation_summary']
        )
    else:
        rng_state_before_baseline = pilot_capture_rng_state()
        r5_checkpoint = torch.load(
            R5_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
        )
        r5_checkpoint_checks = {
            'pilot_version': (
                r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
            ),
            'code_revision': (
                r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
            ),
            'parameter_count': int(
                r5_checkpoint.get('parameter_count', -1)
            ) == EXPECTED_V22_PARAMETERS,
        }
        if not all(r5_checkpoint_checks.values()):
            failed = [
                name for name, passed in r5_checkpoint_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Unsafe r5 read-only baseline checkpoint: '
                + ', '.join(failed)
            )
        r5_baseline_model = VascuFateNetV22(pilot_schema).to(device).eval()
        r5_baseline_model.load_state_dict(r5_checkpoint['ema_model_state'])
        for parameter in r5_baseline_model.parameters():
            parameter.requires_grad_(False)
        print(
            '\nRecomputing the completed r5 best checkpoint with the r7 '
            'precision- and volume-bounded diagnostic contract.'
        )
        r5_recomputed_summary, r5_recomputed_tiers = pilot_validate(
            r5_baseline_model, epoch=0, device=device
        )
        r5_baseline_diagnostic = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'metric_contract_revision': PILOT_CODE_REVISION,
            'r5_pilot_version': R5_PILOT_VERSION,
            'r5_code_revision': R5_PILOT_CODE_REVISION,
            'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'validation_summary': r5_recomputed_summary,
            'component_tier_metrics': r5_recomputed_tiers,
            'r5_checkpoint_loaded_into_r7_model': False,
            'optimizer_steps': 0,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        }
        atomic_write_json(
            r5_baseline_diagnostic, R7_R5_BASELINE_DIAGNOSTIC_PATH
        )
        del r5_baseline_model, r5_checkpoint
        gc.collect()
        torch.cuda.empty_cache()
        pilot_restore_rng_state(rng_state_before_baseline)

    
    
    
    
    del model, ema_model
    gc.collect()
    torch.cuda.empty_cache()
    set_determinism(PILOT_SEED)
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed during lineage reset.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    lineage_smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True
    )
    lineage_smoke_batch = next(iter(lineage_smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        lineage_smoke_loss, lineage_smoke_parts = (
            pilot_compute_training_loss(
                model, lineage_smoke_batch, device, epoch=1, batch_index=0
            )
        )
    if not bool(torch.isfinite(lineage_smoke_loss).item()):
        raise RuntimeError('R7 controlled-lineage smoke loss is non-finite.')
    if float(
        lineage_smoke_parts['component_loss_coefficient']
        .detach().float().cpu()
    ) != 0.0:
        raise RuntimeError(
            'R7 controlled-lineage epoch-1 coefficient is not exactly zero.'
        )
    lineage_smoke_loss.backward()
    lineage_gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            parameter.grad
            for name, parameter in model.named_parameters()
            if name.startswith(prefix) and parameter.grad is not None
        ]
        lineage_gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(gradient).all().item())
            for gradient in gradients
        ) and any(
            float(gradient.detach().float().norm().cpu()) > 0.0
            for gradient in gradients
        )
    model.zero_grad(set_to_none=True)
    if not all(lineage_gradient_checks.values()):
        failed = [
            name for name, passed in lineage_gradient_checks.items()
            if not passed
        ]
        raise RuntimeError(
            'R7 controlled-lineage gradient smoke failed: '
            + ', '.join(failed)
        )

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r7_last_checkpoint_only',
        'strict_best_checkpoint_policy': (
            'save every strict numeric pilot-score improvement'
        ),
        'minimum_delta_policy': (
            'used only to reset early-stopping patience'
        ),
        'r7_preflight_gate_sha256': file_sha256(R7_PREFLIGHT_GATE_PATH),
        'r7_proposal_loss_policy_sha256': file_sha256(R7_POLICY_PATH),
        'component_loss_start_epoch': R7_COMPONENT_LOSS_START_EPOCH,
        'component_loss_full_epoch': R7_COMPONENT_LOSS_FULL_EPOCH,
        'component_loss_maximum_coefficient': (
            R7_COMPONENT_LOSS_MAX_COEFFICIENT
        ),
        'component_positive_weight': R7_POSITIVE_WEIGHT,
        'component_hard_negative_weight': R7_HARD_NEGATIVE_WEIGHT,
        'component_rank_weight': R7_RANK_WEIGHT,
        'component_inner_radius_mm': R7_INNER_RADIUS_MM,
        'component_outer_radius_mm': R7_OUTER_RADIUS_MM,
        'component_logit_margin': R7_LOGIT_MARGIN,
        'component_eligible_tiers': sorted(R7_SMALL_COMPONENT_TIERS),
        'component_coefficient_schedule': {
            str(epoch): coefficient
            for epoch, coefficient in R7_COMPONENT_COEFFICIENT_SCHEDULE.items()
        },
        'proposal_volume_budget_multiplier': R7_VOLUME_BUDGET_MULTIPLIER,
        'proposal_volume_budget_policy': (
            'diagnostic only: retain the top 3x ground-truth target voxels; '
            'never used as an inference rule or checkpoint-selection input'
        ),
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'controlled_lineage_matches_r5_seed': True,
        'controlled_lineage_epoch1_component_coefficient': 0.0,
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'controlled_lineage_gradient_checks': lineage_gradient_checks,
        'controlled_lineage_smoke_loss': float(
            lineage_smoke_loss.detach().cpu()
        ),
        'r7_selected_component_smoke_checks': r7_smoke_checks,
        'r7_component_coefficient_schedule_verified': True,
        'r7_smoke_subject_id': smoke_subject_id,
        'r7_smoke_schedule_index': smoke_index,
        'r7_smoke_dropout_flags': smoke_dropout_flags,
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R7 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R7 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r7 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Controlled r5-seed lineage:          verified')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('Local contrastive rescue:              small tiers, epochs 8-40')
    print('Maximum contrastive coefficient:       0.10')
    print('Proposal anti-shortcut diagnostics:    oracle-volume-bounded (audit only)')
    print('R5 recomputed mean/global Dice:        '
          f'{r5_recomputed_summary["positive_mean_dice"]:.4f} / '
          f'{r5_recomputed_summary["positive_global_dice"]:.4f}')
    print('R5 oracle-budgeted small within 8 mm:  '
          f'{r5_recomputed_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}')
    print('Best-checkpoint minimum delta:        disabled (strict numeric best)')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        strict_best_improved = current_score > best_score
        patience_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if strict_best_improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
        if patience_improved:
            patience_reference_score = current_score
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        epoch_tier_metrics = {
            str(row['component_tier']): row for row in tier_rows
        }
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'proposal_voxel_precision': summary['proposal_voxel_precision'],
            'proposal_predicted_to_target_volume_ratio': summary[
                'proposal_predicted_to_target_volume_ratio'
            ],
            'proposal_voxel_average_precision_256bin': summary[
                'proposal_voxel_average_precision_256bin'
            ],
            'budgeted_proposal_component_recall': summary[
                'budgeted_proposal_component_recall'
            ],
            'small_component_proposal_within_2mm_fraction': summary[
                'small_component_proposal_within_2mm_fraction'
            ],
            'small_component_proposal_within_4mm_fraction': summary[
                'small_component_proposal_within_4mm_fraction'
            ],
            'small_component_proposal_within_8mm_fraction': summary[
                'small_component_proposal_within_8mm_fraction'
            ],
            'small_component_budgeted_proposal_within_8mm_fraction': summary[
                'small_component_budgeted_proposal_within_8mm_fraction'
            ],
            'small_component_representation_failure_fraction': summary[
                'small_component_representation_failure_fraction'
            ],
            **{
                f'{tier_name}_{metric_name}': float(
                    epoch_tier_metrics.get(tier_name, {}).get(
                        metric_name, float('nan')
                    )
                )
                for tier_name in COMPONENT_TIERS
                for metric_name in (
                    'final_component_recall',
                    'proposal_component_recall',
                    'budgeted_proposal_component_recall',
                    'proposal_within_8mm_fraction',
                    'budgeted_proposal_within_8mm_fraction',
                    'representation_failure_fraction',
                )
            },
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'strict_best_improved': strict_best_improved,
            'patience_improved_by_minimum_delta': patience_improved,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if strict_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'contrastive={running["local_contrastive_loss"]/max(processed,1):.4f}, '
            f'coef={running["component_loss_coefficient"]/max(processed,1):.3f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        print(
            '  R7 raw proposal geometry: small within 2/4/8 mm='
            f'{summary["small_component_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_8mm_fraction"]:.4f}, '
            'small representation failure='
            f'{summary["small_component_representation_failure_fraction"]:.4f}'
        )
        print(
            '  Oracle-volume-bounded diagnostic: small within 2/4/8 mm='
            f'{summary["small_component_budgeted_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}, '
            f'component recall={summary["budgeted_proposal_component_recall"]:.4f}, '
            f'precision/AP={summary["proposal_voxel_precision"]:.4f}/'
            f'{summary["proposal_voxel_average_precision_256bin"]:.4f}, '
            f'volume ratio={summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_baseline = r7_policy['r5_baseline']
    success_policy = r7_policy['r7_directional_success_gates']
    best_tier_by_name = {
        str(row['component_tier']): row for row in best_tier_rows
    }
    best_large_proposal_recall = float(
        best_tier_by_name.get('large', {}).get(
            'proposal_component_recall', 0.0
        )
    )
    mean_dice_improvement = (
        float(best_summary['positive_mean_dice'])
        - float(r5_recomputed_summary['positive_mean_dice'])
    )
    budgeted_small_within_8mm_improvement = (
        float(best_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
        - float(r5_recomputed_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
    )
    small_representation_failure_reduction = (
        float(r5_recomputed_summary[
            'small_component_representation_failure_fraction'
        ])
        - float(best_summary[
            'small_component_representation_failure_fraction'
        ])
    )
    global_dice_regression = (
        float(r5_recomputed_summary['positive_global_dice'])
        - float(best_summary['positive_global_dice'])
    )
    proposal_dice_regression = (
        float(r5_recomputed_summary['proposal_global_dice'])
        - float(best_summary['proposal_global_dice'])
    )
    r7_success_checks = {
        'mean_dice_does_not_materially_regress': (
            -mean_dice_improvement
            <= float(success_policy[
                'mean_dice_maximum_allowed_regression'
            ])
        ),
        'mean_dice_reaches_target_improvement': (
            mean_dice_improvement
            >= float(success_policy[
                'mean_dice_target_absolute_improvement'
            ])
        ),
        'volume_bounded_small_within_8mm_improves': (
            budgeted_small_within_8mm_improvement
            >= float(success_policy[
                'volume_bounded_small_within_8mm_absolute_improvement_at_least'
            ])
        ),
        'small_representation_failure_reduced_by_required_margin': (
            small_representation_failure_reduction
            >= float(success_policy[
                'small_representation_failure_absolute_reduction_at_least'
            ])
        ),
        'proposal_component_recall_meets_minimum': (
            float(best_summary['proposal_component_recall'])
            >= float(success_policy['proposal_component_recall_minimum'])
        ),
        'global_dice_regression_within_limit': (
            global_dice_regression
            <= float(success_policy[
                'global_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_dice_regression_within_limit': (
            proposal_dice_regression
            <= float(success_policy[
                'proposal_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_volume_ratio_is_bounded': (
            float(best_summary[
                'proposal_predicted_to_target_volume_ratio'
            ])
            <= float(success_policy[
                'proposal_predicted_to_target_volume_ratio_maximum'
            ])
        ),
        'empty_false_positive_volume_within_limit': (
            float(best_summary['empty_false_positive_volume_ml'])
            <= float(success_policy[
                'mean_false_positive_volume_ml_maximum'
            ])
        ),
        'large_component_proposal_recall_meets_minimum': (
            best_large_proposal_recall
            >= float(success_policy[
                'large_component_proposal_recall_minimum'
            ])
        ),
    }
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'r5_baseline_evidence': r5_baseline,
        'r5_recomputed_under_r7_metric_contract': r5_recomputed_summary,
        'r5_read_only_baseline_diagnostic_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r7_directional_success_policy': success_policy,
        'r7_directional_success_checks': r7_success_checks,
        'r7_directional_success_passed': all(r7_success_checks.values()),
        'r7_directional_effects': {
            'mean_dice_absolute_change': mean_dice_improvement,
            'volume_bounded_small_within_8mm_absolute_change': (
                budgeted_small_within_8mm_improvement
            ),
            'small_representation_failure_absolute_reduction': (
                small_representation_failure_reduction
            ),
            'global_dice_regression': global_dice_regression,
            'proposal_dice_regression': proposal_dice_regression,
            'proposal_predicted_to_target_volume_ratio': float(
                best_summary['proposal_predicted_to_target_volume_ratio']
            ),
            'proposal_voxel_average_precision_256bin': float(
                best_summary['proposal_voxel_average_precision_256bin']
            ),
            'large_component_proposal_recall': (
                best_large_proposal_recall
            ),
        },
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(
        'Raw small proposal within 8 mm:   '
        f'{best_summary["small_component_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Oracle-budgeted small within 8 mm: '
        f'{best_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Proposal precision / AP:          '
        f'{best_summary["proposal_voxel_precision"]:.4f} / '
        f'{best_summary["proposal_voxel_average_precision_256bin"]:.4f}'
    )
    print(
        'Proposal predicted/target ratio:  '
        f'{best_summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
    )
    print(
        'Mean Dice change vs recomputed r5:'
        f' {mean_dice_improvement:+.4f}'
    )
    print(
        'Budgeted within-8-mm change vs r5:'
        f' {budgeted_small_within_8mm_improvement:+.4f}'
    )
    print(
        'Small representation failure:     '
        f'{best_summary["small_component_representation_failure_fraction"]:.4f}'
    )
    print(
        'R7 directional success gates:     '
        f'{all(r7_success_checks.values())}'
    )
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')






R7D_EXPOSURE_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure.csv'
R7D_EXPOSURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure_by_tier.csv'
)
R7D_RESOLUTION_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_component_audit.csv'
)
R7D_RESOLUTION_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_summary.csv'
)
R7D_PAIRED_CASE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_case_metrics.csv'
)
R7D_PAIRED_COMPONENT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_component_metrics.csv'
)
R7D_MARGIN_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_train_heldout_margin_audit.csv'
)
R7D_MARGIN_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_margin_summary.csv'
)
R7D_VOLUME_BUDGET_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_audit.csv'
)
R7D_VOLUME_BUDGET_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_summary.csv'
)
R7D_CHECKPOINT_PARETO_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'checkpoint_metric_pareto.csv'
)
R7D_BRANCH_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'branch_reliance_comparison.csv'
)
R7D_VISUAL_CANDIDATES_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'visual_qa_candidates.csv'
)
R7D_CHECKS_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'diagnostic_checks.csv'
R7D_TARGET_ACCESS_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'development_target_access_audit.csv'
)

R7D_OUTPUT_PATHS = [
    R7D_EXPOSURE_PATH,
    R7D_EXPOSURE_SUMMARY_PATH,
    R7D_RESOLUTION_PATH,
    R7D_RESOLUTION_SUMMARY_PATH,
    R7D_PAIRED_CASE_PATH,
    R7D_PAIRED_COMPONENT_PATH,
    R7D_MARGIN_PATH,
    R7D_MARGIN_SUMMARY_PATH,
    R7D_VOLUME_BUDGET_PATH,
    R7D_VOLUME_BUDGET_SUMMARY_PATH,
    R7D_CHECKPOINT_PARETO_PATH,
    R7D_BRANCH_PATH,
    R7D_VISUAL_CANDIDATES_PATH,
    R7D_CHECKS_PATH,
    R7D_TARGET_ACCESS_PATH,
    R7_FAILURE_SUMMARY_PATH,
]


def r7d_existing_result_is_complete():
    if not R7_FAILURE_GATE_PATH.exists():
        return False
    gate = load_json(R7_FAILURE_GATE_PATH)
    if not (
        gate.get('diagnostic_complete') is True
        and gate.get('diagnostic_version') == R7_FAILURE_DIAGNOSTIC_VERSION
        and gate.get('code_revision') == R7_FAILURE_DIAGNOSTIC_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_checkpoint_sha256')
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    ):
        return False
    output_hashes = gate.get('output_sha256', {})
    return all(
        path.exists()
        and output_hashes.get(path.name) == file_sha256(path)
        for path in R7D_OUTPUT_PATHS
    )


def r7d_gini(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0 or float(values.sum()) <= 0.0:
        return 0.0
    values = np.sort(values)
    index = np.arange(1, values.size + 1, dtype=np.float64)
    return float(
        np.sum((2.0 * index - values.size - 1.0) * values)
        / (values.size * values.sum())
    )


def r7d_component_support(row):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    mode = str(row['trial8_supervision_mode'])
    if mode == 'hard_and_soft':
        support = binary
    elif mode == 'soft_only_subvoxel':
        support = soft > 0
    elif mode == 'true_empty':
        support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unsupported supervision mode {mode!r}.'
        )
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: non-finite target arrays.')
    return binary, soft, support, spacing


def r7d_component_schedule_exposure():
    counts = defaultdict(int)
    first_epoch = {}
    last_epoch = {}
    for epoch in range(R7_COMPONENT_LOSS_START_EPOCH, MAX_EPOCHS + 1):
        schedule = pilot_build_epoch_schedule(epoch)
        for entry in schedule:
            if (
                str(entry['sample_type']) != 'positive'
                or str(entry['component_tier']) not in R7_SMALL_COMPONENT_TIERS
            ):
                continue
            key = (
                str(entry['subject_id']), int(entry['atomic_component_id'])
            )
            counts[key] += 1
            first_epoch.setdefault(key, int(epoch))
            last_epoch[key] = int(epoch)

    rows = []
    for item in component_df.sort_values(
        ['subject_id', 'atomic_component_id']
    ).to_dict('records'):
        key = (str(item['subject_id']), int(item['atomic_component_id']))
        tier = str(item['component_tier'])
        eligible = tier in R7_SMALL_COMPONENT_TIERS
        selections = int(counts.get(key, 0)) if eligible else 0
        rows.append({
            'subject_id': key[0],
            'atomic_component_id': key[1],
            'component_tier': tier,
            'component_voxels': int(item['component_voxels']),
            'component_volume_ml': float(item['component_volume_ml']),
            'global_proposal_voxels': int(item['global_proposal_voxels']),
            'local_contrastive_eligible': bool(eligible),
            'epochs_eligible': (
                MAX_EPOCHS - R7_COMPONENT_LOSS_START_EPOCH + 1
                if eligible else 0
            ),
            'selections_epochs_8_to_40': selections,
            'first_selected_epoch': first_epoch.get(key, np.nan),
            'last_selected_epoch': last_epoch.get(key, np.nan),
            'never_selected': bool(eligible and selections == 0),
            'selected_once': bool(eligible and selections == 1),
            'selected_multiple_times': bool(eligible and selections > 1),
        })
    frame = pd.DataFrame(rows)
    summary_rows = []
    for tier in COMPONENT_TIERS:
        group = frame[frame['component_tier'].astype(str) == tier]
        eligible = group[group['local_contrastive_eligible'].astype(bool)]
        values = eligible['selections_epochs_8_to_40'].to_numpy(
            dtype=np.float64
        )
        summary_rows.append({
            'component_tier': tier,
            'components': int(len(group)),
            'eligible_components': int(len(eligible)),
            'never_selected_components': int(
                eligible['never_selected'].sum()
            ),
            'never_selected_fraction': float(
                eligible['never_selected'].mean()
            ) if len(eligible) else 0.0,
            'selected_once_fraction': float(
                eligible['selected_once'].mean()
            ) if len(eligible) else 0.0,
            'minimum_selections': int(values.min()) if len(values) else 0,
            'median_selections': float(np.median(values)) if len(values) else 0.0,
            'mean_selections': float(values.mean()) if len(values) else 0.0,
            'maximum_selections': int(values.max()) if len(values) else 0,
            'selection_gini': r7d_gini(values),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(frame, R7D_EXPOSURE_PATH)
    atomic_write_csv(summary, R7D_EXPOSURE_SUMMARY_PATH)
    return frame, summary


def r7d_maximum_component_cell_matching(cell_sets):
    matched_cell = {}

    def visit(component_index, seen):
        for cell_id in sorted(cell_sets[component_index]):
            if cell_id in seen:
                continue
            seen.add(cell_id)
            if (
                cell_id not in matched_cell
                or visit(matched_cell[cell_id], seen)
            ):
                matched_cell[cell_id] = component_index
                return True
        return False

    matched = 0
    for component_index in range(len(cell_sets)):
        matched += int(visit(component_index, set()))
    return int(matched)


def r7d_resolution_audit():
    component_rows = []
    subject_rows = []
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    canvas_shape = np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    global_shape = tuple(int(v) for v in GLOBAL_INPUT_SHAPE)
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            if partition == 'fold0_fit':
                expected = len(component_records.get(subject_id, []))
                if int(count) != int(expected):
                    raise RuntimeError(
                        f'{subject_id}: resolution audit reconstructed '
                        f'{count} components; frozen audit has {expected}.'
                    )
            pad_start = (canvas_shape - np.asarray(support.shape)) // 2
            cell_sets = []
            component_metadata = []
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for component_id in range(1, int(count) + 1):
                coordinates = np.argwhere(labels == component_id)
                pooled = np.unique(
                    (coordinates + pad_start) // downsample, axis=0
                )
                cell_ids = set(
                    np.ravel_multi_index(pooled.T, global_shape).tolist()
                )
                cell_sets.append(cell_ids)
                voxels = int(len(coordinates))
                volume_ml = voxels * voxel_volume_ml
                component_metadata.append((voxels, volume_ml, pooled))
            cell_frequency = defaultdict(int)
            signature_frequency = defaultdict(int)
            for cells in cell_sets:
                for cell_id in cells:
                    cell_frequency[cell_id] += 1
                signature_frequency[tuple(sorted(cells))] += 1
            matched = r7d_maximum_component_cell_matching(cell_sets)
            collision_components = 0
            duplicate_signature_components = 0
            for component_index, cells in enumerate(cell_sets):
                voxels, volume_ml, pooled = component_metadata[component_index]
                shared_cells = sum(
                    int(cell_frequency[cell_id] > 1) for cell_id in cells
                )
                duplicate_signature = (
                    signature_frequency[tuple(sorted(cells))] > 1
                )
                collision = shared_cells > 0
                collision_components += int(collision)
                duplicate_signature_components += int(duplicate_signature)
                component_rows.append({
                    'development_partition': partition,
                    'subject_id': subject_id,
                    'component_id': int(component_index + 1),
                    'component_tier': pilot_component_tier(volume_ml, voxels),
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'global_proposal_voxels': int(len(cells)),
                    'single_global_voxel': bool(len(cells) == 1),
                    'shared_global_voxels': int(shared_cells),
                    'shares_any_global_voxel': bool(collision),
                    'duplicate_pooled_signature': bool(duplicate_signature),
                    'native_voxels_per_global_voxel': float(
                        voxels / max(len(cells), 1)
                    ),
                })
            subject_rows.append({
                'development_partition': partition,
                'subject_id': subject_id,
                'components': int(count),
                'distinct_global_voxels_used': int(len(cell_frequency)),
                'components_sharing_any_global_voxel': int(collision_components),
                'components_with_duplicate_pooled_signature': int(
                    duplicate_signature_components
                ),
                'maximum_distinct_component_assignment': int(matched),
                'distinct_assignment_ceiling_fraction': float(
                    matched / max(int(count), 1)
                ) if count else 1.0,
            })
            if case_index in {1, len(frame)} or case_index % 20 == 0:
                print(
                    f'  Resolution {partition} {case_index:02d}/{len(frame)}: '
                    f'{subject_id}, components={count}'
                )
    components = pd.DataFrame(component_rows)
    subjects = pd.DataFrame(subject_rows)
    summary_rows = []
    for partition in ['fold0_fit', 'fold0_heldout']:
        for tier in COMPONENT_TIERS:
            group = components[
                (components['development_partition'] == partition)
                & (components['component_tier'] == tier)
            ]
            summary_rows.append({
                'development_partition': partition,
                'component_tier': tier,
                'components': int(len(group)),
                'single_global_voxel_fraction': float(
                    group['single_global_voxel'].mean()
                ) if len(group) else 0.0,
                'global_voxel_collision_fraction': float(
                    group['shares_any_global_voxel'].mean()
                ) if len(group) else 0.0,
                'duplicate_pooled_signature_fraction': float(
                    group['duplicate_pooled_signature'].mean()
                ) if len(group) else 0.0,
                'median_global_proposal_voxels': float(
                    group['global_proposal_voxels'].median()
                ) if len(group) else 0.0,
                'mean_distinct_assignment_ceiling_fraction': float('nan'),
            })
    for partition, group in subjects.groupby('development_partition'):
        summary_rows.append({
            'development_partition': str(partition),
            'component_tier': 'subject_level_distinct_assignment_ceiling',
            'components': int(group['components'].sum()),
            'single_global_voxel_fraction': float('nan'),
            'global_voxel_collision_fraction': float('nan'),
            'duplicate_pooled_signature_fraction': float('nan'),
            'median_global_proposal_voxels': float('nan'),
            'mean_distinct_assignment_ceiling_fraction': float(
                group['distinct_assignment_ceiling_fraction'].mean()
            ),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(components, R7D_RESOLUTION_PATH)
    atomic_write_csv(summary, R7D_RESOLUTION_SUMMARY_PATH)
    return components, subjects, summary


def r7d_validation_artifacts(epoch):
    root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    paths = {
        'case': root / 'case_metrics.csv',
        'component': root / 'component_metrics.csv',
        'tier': root / 'component_tier_metrics.csv',
        'summary': root / 'validation_summary.json',
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise RuntimeError(
            'Completed validation artifacts are missing: ' + ', '.join(missing)
        )
    return paths


def r7d_paired_completed_metrics(r7_best_epoch):
    r5_paths = r7d_validation_artifacts(0)
    r7_paths = r7d_validation_artifacts(r7_best_epoch)
    r5_case = pd.read_csv(r5_paths['case'])
    r7_case = pd.read_csv(r7_paths['case'])
    case_keys = ['subject_id']
    paired_case = r5_case.merge(
        r7_case, on=case_keys, how='outer', suffixes=('_r5', '_r7'),
        validate='one_to_one', indicator=True,
    )
    if not bool((paired_case['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 case metrics do not pair one-to-one.')
    paired_case = paired_case.drop(columns=['_merge'])
    case_delta_columns = [
        'dice', 'voxel_precision', 'voxel_recall',
        'predicted_volume_ml_from_mask', 'predicted_volume_ml_from_router',
        'presence_probability', 'proposal_voxel_precision',
        'proposal_voxel_recall', 'proposal_predicted_to_target_volume_ratio',
    ]
    for name in case_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_case and right in paired_case:
            paired_case[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_case[right], errors='coerce')
                - pd.to_numeric(paired_case[left], errors='coerce')
            )

    r5_component = pd.read_csv(r5_paths['component'])
    r7_component = pd.read_csv(r7_paths['component'])
    component_keys = ['subject_id', 'component_id']
    paired_component = r5_component.merge(
        r7_component, on=component_keys, how='outer',
        suffixes=('_r5', '_r7'), validate='one_to_one', indicator=True,
    )
    if not bool((paired_component['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 component metrics do not pair one-to-one.')
    paired_component = paired_component.drop(columns=['_merge'])
    component_delta_columns = [
        'final_probability_maximum', 'proposal_probability_maximum',
        'final_maximum_component_iou', 'proposal_maximum_component_iou',
        'proposal_minimum_distance_mm',
        'budgeted_proposal_maximum_component_iou',
        'budgeted_proposal_minimum_distance_mm',
    ]
    for name in component_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_component[right], errors='coerce')
                - pd.to_numeric(paired_component[left], errors='coerce')
            )
    for name in [
        'final_detected', 'proposal_covered',
        'budgeted_proposal_within_8mm',
        'representation_failure_below_0p10',
    ]:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                paired_component[right].map(as_bool).astype(int)
                - paired_component[left].map(as_bool).astype(int)
            )
    atomic_write_csv(paired_case, R7D_PAIRED_CASE_PATH)
    atomic_write_csv(paired_component, R7D_PAIRED_COMPONENT_PATH)

    candidates = []
    positive_case_candidates = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    for label, ascending in [
        ('largest_case_dice_regression', True),
        ('largest_case_dice_improvement', False),
    ]:
        ranked = positive_case_candidates.sort_values(
            'dice_delta_r7_minus_r5', ascending=ascending
        ).head(5)
        for rank, item in enumerate(ranked.to_dict('records'), start=1):
            candidates.append({
                'candidate_reason': label,
                'rank': rank,
                'subject_id': str(item['subject_id']),
                'component_id': np.nan,
                'component_tier': '',
                'r5_value': float(item['dice_r5']),
                'r7_value': float(item['dice_r7']),
                'delta_r7_minus_r5': float(
                    item['dice_delta_r7_minus_r5']
                ),
            })
    small = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ].copy()
    small = small.sort_values(
        'proposal_probability_maximum_delta_r7_minus_r5', ascending=True
    ).head(10)
    for rank, item in enumerate(small.to_dict('records'), start=1):
        candidates.append({
            'candidate_reason': 'largest_small_proposal_probability_regression',
            'rank': rank,
            'subject_id': str(item['subject_id']),
            'component_id': int(item['component_id']),
            'component_tier': str(item['component_tier_r7']),
            'r5_value': float(item['proposal_probability_maximum_r5']),
            'r7_value': float(item['proposal_probability_maximum_r7']),
            'delta_r7_minus_r5': float(
                item['proposal_probability_maximum_delta_r7_minus_r5']
            ),
        })
    candidate_frame = pd.DataFrame(candidates)
    atomic_write_csv(candidate_frame, R7D_VISUAL_CANDIDATES_PATH)
    return paired_case, paired_component, candidate_frame


def r7d_checkpoint_pareto(r7_best_epoch):
    history = pd.read_csv(PILOT_HISTORY_PATH).sort_values('epoch').copy()
    maximize = [
        'validation_mean_dice',
        'validation_global_dice',
        'validation_small_mean_dice',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'proposal_voxel_average_precision_256bin',
    ]
    maximize = [name for name in maximize if name in history.columns]
    values = history[maximize].apply(pd.to_numeric, errors='coerce').fillna(
        -np.inf
    ).to_numpy(dtype=np.float64)
    pareto = np.ones(len(history), dtype=bool)
    for index in range(len(history)):
        dominated = np.any(
            np.all(values >= values[index], axis=1)
            & np.any(values > values[index], axis=1)
        )
        pareto[index] = not bool(dominated)
    history['pareto_nondominated'] = pareto
    history['is_selected_best_epoch'] = (
        history['epoch'].astype(int) == int(r7_best_epoch)
    )
    best_rows = {}
    metric_flags = {}
    for name in maximize + ['pilot_checkpoint_score']:
        if name not in history:
            continue
        values_series = pd.to_numeric(history[name], errors='coerce')
        best_index = values_series.idxmax()
        best_epoch = int(history.loc[best_index, 'epoch'])
        best_value = float(values_series.loc[best_index])
        flag = f'is_best_{name}'
        history[flag] = history['epoch'].astype(int) == best_epoch
        best_rows[name] = {'epoch': best_epoch, 'value': best_value}
        metric_flags[name] = flag
    atomic_write_csv(history, R7D_CHECKPOINT_PARETO_PATH)
    return history, best_rows


def r7d_branch_comparison(r5_best_epoch, r7_best_epoch):
    sources = [
        ('r5', R5_PILOT_ROOT / 'branch_reliance_history.csv', r5_best_epoch),
        ('r7', PILOT_BRANCH_HISTORY_PATH, r7_best_epoch),
    ]
    selected_frames = []
    for checkpoint, path, epoch in sources:
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        available_epochs = sorted(frame['epoch'].astype(int).unique().tolist())
        if not available_epochs:
            continue
        selected_epoch = (
            int(epoch) if int(epoch) in available_epochs
            else min(available_epochs, key=lambda value: abs(value - int(epoch)))
        )
        selected = frame[frame['epoch'].astype(int) == selected_epoch].copy()
        selected['checkpoint'] = checkpoint
        selected['requested_best_epoch'] = int(epoch)
        selected['selected_probe_epoch'] = int(selected_epoch)
        selected['exact_best_epoch_probe'] = bool(selected_epoch == int(epoch))
        selected_frames.append(selected)
    if not selected_frames:
        comparison = pd.DataFrame(columns=[
            'checkpoint', 'ablation', 'mean_target_probability_delta'
        ])
    else:
        raw = pd.concat(selected_frames, ignore_index=True)
        rows = []
        for (checkpoint, ablation), group in raw.groupby(
            ['checkpoint', 'ablation']
        ):
            rows.append({
                'checkpoint': str(checkpoint),
                'ablation': str(ablation),
                'subjects': int(group['subject_id'].nunique()),
                'requested_best_epoch': int(
                    group['requested_best_epoch'].iloc[0]
                ),
                'selected_probe_epoch': int(
                    group['selected_probe_epoch'].iloc[0]
                ),
                'exact_best_epoch_probe': bool(
                    group['exact_best_epoch_probe'].iloc[0]
                ),
                'mean_target_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_target_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_local_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_local_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_proposal_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_proposal_probability_delta'],
                    errors='coerce',
                ).mean()),
            })
        comparison = pd.DataFrame(rows)
        if set(comparison['checkpoint']) == {'r5', 'r7'}:
            pivot = comparison.pivot(
                index='ablation', columns='checkpoint',
                values='mean_target_probability_delta',
            )
            deltas = (pivot['r7'] - pivot['r5']).to_dict()
            comparison['target_delta_change_r7_minus_r5'] = comparison[
                'ablation'
            ].map(deltas)
    atomic_write_csv(comparison, R7D_BRANCH_PATH)
    return comparison


def r7d_global_component_masks(labels, count, pad_start):
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pad = np.asarray(pad_start, dtype=np.int64)
    all_coordinates = np.argwhere(labels > 0)
    full = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    if len(all_coordinates):
        pooled = np.unique((all_coordinates + pad) // downsample, axis=0)
        full[tuple(pooled.T)] = True
    components = []
    for component_id in range(1, int(count) + 1):
        coordinates = np.argwhere(labels == component_id)
        pooled = np.unique((coordinates + pad) // downsample, axis=0)
        components.append(pooled)
    return full, components


def r7d_global_domain(native_shape, pad_start):
    start = np.asarray(pad_start, dtype=np.int64)
    end = start + np.asarray(native_shape, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    low_start = start // downsample
    low_end = (end + downsample - 1) // downsample
    domain = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    slices = tuple(
        slice(int(low_start[axis]), int(low_end[axis])) for axis in range(3)
    )
    domain[slices] = True
    return domain


def r7d_component_logit_margins(
    logits, full_target, component_coordinates, domain, subject_id,
    partition, checkpoint, spacing,
):
    rows = []
    outer_radius = np.asarray(R7_OUTER_STRUCTURE.shape, dtype=np.int64) // 2
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    native_voxel_volume = voxel_volume_ml
    for component_id, pooled in enumerate(component_coordinates, start=1):
        if len(pooled) == 0:
            continue
        minimum = np.maximum(pooled.min(axis=0) - outer_radius, 0)
        maximum = np.minimum(
            pooled.max(axis=0) + outer_radius + 1,
            np.asarray(GLOBAL_INPUT_SHAPE),
        )
        slices = tuple(
            slice(int(minimum[axis]), int(maximum[axis]))
            for axis in range(3)
        )
        exact = np.zeros(tuple((maximum - minimum).tolist()), dtype=bool)
        local_coordinates = pooled - minimum
        exact[tuple(local_coordinates.T)] = True
        _, _, negative = r7_component_regions(
            exact, full_target[slices], domain[slices]
        )
        positive_logits = logits[slices][exact]
        negative_logits = logits[slices][negative]
        hard_count = min(
            len(negative_logits),
            max(R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * len(positive_logits)),
        )
        if hard_count > 0:
            hard_negative = np.partition(
                negative_logits, len(negative_logits) - hard_count
            )[-hard_count:]
            hard_negative_mean = float(hard_negative.mean())
            hard_negative_maximum = float(hard_negative.max())
        else:
            hard_negative_mean = float('nan')
            hard_negative_maximum = float('nan')
        native_voxels_estimate = int(len(pooled) * np.prod(GLOBAL_DOWNSAMPLE))
        volume_ml_estimate = native_voxels_estimate * native_voxel_volume
        positive_mean = float(positive_logits.mean())
        positive_minimum = float(positive_logits.min())
        positive_maximum = float(positive_logits.max())
        rows.append({
            'checkpoint': checkpoint,
            'development_partition': partition,
            'subject_id': subject_id,
            'component_id': int(component_id),
            'component_tier_from_pooled_extent': pilot_component_tier(
                volume_ml_estimate, native_voxels_estimate
            ),
            'global_proposal_voxels': int(len(pooled)),
            'positive_logit_mean': positive_mean,
            'positive_logit_minimum': positive_minimum,
            'positive_logit_maximum': positive_maximum,
            'hard_negative_logit_mean': hard_negative_mean,
            'hard_negative_logit_maximum': hard_negative_maximum,
            'positive_minus_hard_negative_mean_margin': (
                positive_mean - hard_negative_mean
                if np.isfinite(hard_negative_mean) else float('nan')
            ),
            'positive_minimum_minus_hard_negative_maximum_margin': (
                positive_minimum - hard_negative_maximum
                if np.isfinite(hard_negative_maximum) else float('nan')
            ),
            'positive_probability_mean': float(
                1.0 / (1.0 + np.exp(-np.clip(positive_mean, -30, 30)))
            ),
            'negative_annulus_voxels': int(negative.sum()),
            'hard_negative_voxels_used': int(hard_count),
        })
    return rows


def r7d_topk_prediction(probability, budget):
    probability = np.asarray(probability, dtype=np.float32)
    budget = int(np.clip(int(budget), 1, probability.size))
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape)


def r7d_budget_metrics(
    probability, target, spacing, predicted_volume_ml, subject_id,
    checkpoint,
):
    target = np.asarray(target, dtype=bool)
    target_voxels = int(target.sum())
    if target_voxels == 0:
        return []
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    labels, count = ndimage.label(
        target, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    component_objects = ndimage.find_objects(labels)
    component_metadata = []
    for component_id, component_slice in enumerate(component_objects, start=1):
        if component_slice is None:
            continue
        local = labels[component_slice] == component_id
        voxels = int(local.sum())
        volume_ml = voxels * voxel_volume_ml
        component_metadata.append((
            component_id, component_slice, local,
            pilot_component_tier(volume_ml, voxels),
        ))
    budgets = [
        ('router_x1', max(1, int(math.ceil(predicted_volume_ml / voxel_volume_ml)))),
        ('router_x2', max(1, int(math.ceil(2.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('router_x3', max(1, int(math.ceil(3.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('oracle_target_x3', max(1, int(math.ceil(3.0 * target_voxels)))),
    ]
    rows = []
    for budget_name, requested_budget in budgets:
        prediction = r7d_topk_prediction(probability, requested_budget)
        predicted_voxels = int(prediction.sum())
        intersection = int((prediction & target).sum())
        distance = ndimage.distance_transform_edt(
            ~prediction, sampling=spacing
        )
        within_8 = []
        small_within_8 = []
        for _, component_slice, component_local, tier in component_metadata:
            minimum_distance = float(
                distance[component_slice][component_local].min(initial=np.inf)
            )
            covered = bool(minimum_distance <= 8.0)
            within_8.append(covered)
            if tier in R7_SMALL_COMPONENT_TIERS:
                small_within_8.append(covered)
        rows.append({
            'checkpoint': checkpoint,
            'subject_id': subject_id,
            'budget_policy': budget_name,
            'router_predicted_volume_ml': float(predicted_volume_ml),
            'target_volume_ml': float(target_voxels * voxel_volume_ml),
            'router_to_target_volume_ratio': float(
                predicted_volume_ml / max(target_voxels * voxel_volume_ml, 1e-8)
            ),
            'requested_budget_voxels': int(requested_budget),
            'selected_budget_voxels': predicted_voxels,
            'selected_to_target_voxel_ratio': float(
                predicted_voxels / max(target_voxels, 1)
            ),
            'voxel_precision': float(intersection / max(predicted_voxels, 1)),
            'voxel_recall': float(intersection / max(target_voxels, 1)),
            'dice': float(
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            ),
            'components': int(len(component_metadata)),
            'component_within_8mm_fraction': float(np.mean(within_8)),
            'small_components': int(len(small_within_8)),
            'small_component_within_8mm_fraction': float(
                np.mean(small_within_8)
            ) if small_within_8 else float('nan'),
        })
    return rows


def r7d_load_model(checkpoint_path, expected_version, expected_revision, device):
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=False
    )
    checks = {
        'pilot_version': checkpoint.get('pilot_version') == expected_version,
        'code_revision': checkpoint.get('code_revision') == expected_revision,
        'parameter_count': int(checkpoint.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS,
        'ema_model_state': 'ema_model_state' in checkpoint,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'Unsafe read-only checkpoint {checkpoint_path}: '
            + ', '.join(failed)
        )
    model = VascuFateNetV22(pilot_schema).to(device)
    model.load_state_dict(checkpoint['ema_model_state'], strict=True)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    del checkpoint
    return model


@torch.no_grad()
def r7d_checkpoint_margin_and_volume_audit(
    checkpoint_label, checkpoint_path, expected_version,
    expected_revision, device,
):
    model = r7d_load_model(
        checkpoint_path, expected_version, expected_revision, device
    )
    margin_rows = []
    budget_rows = []
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            case = pilot_load_validation_case(row)
            batch = pilot_stack_sample(case['base_sample'], device)
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(model, batch)
            proposal_logits_tensor = cache['proposal_logits'].float()
            logits = proposal_logits_tensor[0, 0].cpu().numpy()
            predicted_volume_ml = float(torch.expm1(
                cache['routed']['log_volume_prediction'].float()
            )[0].clamp_min(0).cpu())
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            full_global, component_coordinates = r7d_global_component_masks(
                labels, count, case['pad_start']
            )
            domain = r7d_global_domain(case['native_shape'], case['pad_start'])
            rows = r7d_component_logit_margins(
                logits, full_global, component_coordinates, domain,
                subject_id, partition, checkpoint_label, spacing,
            )
            
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for item in rows:
                component_id = int(item['component_id'])
                voxels = int((labels == component_id).sum())
                volume_ml = voxels * voxel_volume_ml
                item['component_voxels'] = voxels
                item['component_volume_ml'] = volume_ml
                item['component_tier'] = pilot_component_tier(
                    volume_ml, voxels
                )
            margin_rows.extend(rows)

            if (
                partition == 'fold0_heldout'
                and as_bool(row['trial8_hard_metric_eligible'])
                and bool(case['target'].any())
            ):
                proposal_canvas = F.interpolate(
                    torch.sigmoid(proposal_logits_tensor),
                    size=GLOBAL_CANVAS_SHAPE,
                    mode='trilinear', align_corners=False,
                )[0, 0].cpu().numpy()
                native_slices = tuple(
                    slice(
                        case['pad_start'][axis],
                        case['pad_start'][axis] + case['native_shape'][axis],
                    )
                    for axis in range(3)
                )
                proposal_native = proposal_canvas[native_slices]
                budget_rows.extend(r7d_budget_metrics(
                    proposal_native, case['target'], case['spacing'],
                    predicted_volume_ml, subject_id, checkpoint_label,
                ))
            del batch, cache, proposal_logits_tensor, case
            if case_index in {1, len(frame)} or case_index % 10 == 0:
                print(
                    f'  {checkpoint_label} {partition} '
                    f'{case_index:02d}/{len(frame)}: {subject_id}'
                )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return margin_rows, budget_rows


def r7d_margin_summary(frame):
    rows = []
    for (checkpoint, partition, tier), group in frame.groupby([
        'checkpoint', 'development_partition', 'component_tier'
    ]):
        margin = pd.to_numeric(
            group['positive_minus_hard_negative_mean_margin'],
            errors='coerce',
        )
        rows.append({
            'checkpoint': str(checkpoint),
            'development_partition': str(partition),
            'component_tier': str(tier),
            'components': int(len(group)),
            'mean_logit_margin': float(margin.mean()),
            'median_logit_margin': float(margin.median()),
            'positive_margin_fraction': float((margin > 0).mean()),
            'mean_positive_probability': float(pd.to_numeric(
                group['positive_probability_mean'], errors='coerce'
            ).mean()),
        })
    return pd.DataFrame(rows)


def r7d_volume_budget_summary(frame):
    rows = []
    for (checkpoint, policy), group in frame.groupby(
        ['checkpoint', 'budget_policy']
    ):
        rows.append({
            'checkpoint': str(checkpoint),
            'budget_policy': str(policy),
            'subjects': int(group['subject_id'].nunique()),
            'mean_dice': float(group['dice'].mean()),
            'mean_voxel_precision': float(group['voxel_precision'].mean()),
            'mean_voxel_recall': float(group['voxel_recall'].mean()),
            'mean_component_within_8mm_fraction': float(
                group['component_within_8mm_fraction'].mean()
            ),
            'mean_small_component_within_8mm_fraction': float(
                group['small_component_within_8mm_fraction'].mean()
            ),
            'median_router_to_target_volume_ratio': float(
                group['router_to_target_volume_ratio'].median()
            ),
            'mean_selected_to_target_voxel_ratio': float(
                group['selected_to_target_voxel_ratio'].mean()
            ),
        })
    return pd.DataFrame(rows)


def r7d_small_margin_delta(summary, partition):
    subset = summary[
        (summary['development_partition'] == partition)
        & summary['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    if not len(subset):
        return float('nan')
    weighted = {}
    for checkpoint in ['r5', 'r7']:
        group = subset[subset['checkpoint'] == checkpoint]
        if not len(group):
            weighted[checkpoint] = float('nan')
            continue
        weighted[checkpoint] = float(np.average(
            group['mean_logit_margin'], weights=group['components']
        ))
    return weighted['r7'] - weighted['r5']


def run_r7_failure_mechanism_diagnostic():
    if r7d_existing_result_is_complete():
        gate = load_json(R7_FAILURE_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC ALREADY COMPLETE')
        print('=' * 80)
        print(f'Dominant mechanism: {gate["dominant_failure_mechanism"]}')
        print(f'Diagnostic gate:    {R7_FAILURE_GATE_PATH}')
        print('No source checkpoint or completed diagnostic was rewritten.')
        return

    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r7_best_epoch = int(r7_result['best_epoch'])
    r5_best_epoch = int(r5_result['best_epoch'])
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_sha_before = file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    source_paths = [
        R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH,
        R5_RESULT_GATE_PATH, PILOT_RESULT_GATE_PATH, PILOT_HISTORY_PATH,
    ]
    source_hashes_before = {
        str(path): file_sha256(path) for path in source_paths
    }
    initial_checks = {
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r7_pilot_complete': r7_result.get('pilot_complete') is True,
        'r5_checkpoint_hash_matches_result_gate': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'r7_checkpoint_hash_matches_result_gate': (
            r7_result.get('best_checkpoint_sha256') == r7_sha_before
        ),
        'r7_directional_success_was_false': (
            r7_result.get('r7_directional_success_passed') is False
        ),
        'full_fivefold_training_locked': (
            r7_result.get('full_fivefold_training_authorized') is False
        ),
        'fold0_subject_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
        ),
        'sealed_test_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
        ),
    }
    if not all(initial_checks.values()):
        failed = [name for name, passed in initial_checks.items() if not passed]
        raise RuntimeError(
            'R7 failure diagnostic upstream checks failed: '
            + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC STARTING')
    print('=' * 80)
    print(f'R5 / r7 best epochs:               {r5_best_epoch} / {r7_best_epoch}')
    print('Target access:                     fold-0 development only')
    print('Locked validation targets read:    False')
    print('Test targets read:                 False')
    print('Optimizer / backward / steps:      0 / 0 / 0')

    target_access_frame = pd.DataFrame([
        {
            'subject_id': str(row['subject_id']),
            'development_partition': partition,
            'target_arrays_authorized': True,
            'target_arrays_read_by_diagnostic': True,
            'locked_validation_target_read': False,
            'sealed_test_target_read': False,
        }
        for partition, frame in [
            ('fold0_fit', fold0_fit_df),
            ('fold0_heldout', fold0_heldout_df),
        ]
        for row in frame.to_dict('records')
    ])
    atomic_write_csv(target_access_frame, R7D_TARGET_ACCESS_PATH)

    exposure, exposure_summary = r7d_component_schedule_exposure()
    resolution, resolution_subjects, resolution_summary = (
        r7d_resolution_audit()
    )
    paired_case, paired_component, visual_candidates = (
        r7d_paired_completed_metrics(r7_best_epoch)
    )
    checkpoint_pareto, metric_best_rows = r7d_checkpoint_pareto(
        r7_best_epoch
    )
    branch_comparison = r7d_branch_comparison(
        r5_best_epoch, r7_best_epoch
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            'The train-versus-held-out r5/r7 margin audit requires CUDA.'
        )
    set_determinism(PILOT_SEED)
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    all_margin_rows = []
    all_budget_rows = []
    checkpoint_specs = [
        ('r5', R5_BEST_CHECKPOINT_PATH,
         R5_PILOT_VERSION, R5_PILOT_CODE_REVISION),
        ('r7', PILOT_BEST_CHECKPOINT_PATH,
         PILOT_VERSION, PILOT_CODE_REVISION),
    ]
    for label, path, version, revision in checkpoint_specs:
        print(f'\nRead-only global proposal audit: {label}')
        margin_rows, budget_rows = r7d_checkpoint_margin_and_volume_audit(
            label, path, version, revision, device
        )
        all_margin_rows.extend(margin_rows)
        all_budget_rows.extend(budget_rows)
    margins = pd.DataFrame(all_margin_rows)
    budgets = pd.DataFrame(all_budget_rows)
    margin_summary = r7d_margin_summary(margins)
    budget_summary = r7d_volume_budget_summary(budgets)
    atomic_write_csv(margins, R7D_MARGIN_PATH)
    atomic_write_csv(margin_summary, R7D_MARGIN_SUMMARY_PATH)
    atomic_write_csv(budgets, R7D_VOLUME_BUDGET_PATH)
    atomic_write_csv(budget_summary, R7D_VOLUME_BUDGET_SUMMARY_PATH)

    small_exposure = exposure[
        exposure['local_contrastive_eligible'].astype(bool)
    ]
    never_selected_fraction = float(
        small_exposure['never_selected'].mean()
    )
    heldout_small_resolution = resolution[
        (resolution['development_partition'] == 'fold0_heldout')
        & resolution['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    heldout_small_collision_fraction = float(
        heldout_small_resolution['shares_any_global_voxel'].mean()
    ) if len(heldout_small_resolution) else float('nan')
    heldout_assignment_ceiling = float(
        resolution_subjects[
            resolution_subjects['development_partition'] == 'fold0_heldout'
        ]['distinct_assignment_ceiling_fraction'].mean()
    )
    fit_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_fit'
    )
    heldout_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_heldout'
    )
    positive_paired_cases = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    case_dice_delta = float(
        positive_paired_cases['dice_delta_r7_minus_r5'].mean()
    )
    small_components = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ]
    small_proposal_probability_delta = float(pd.to_numeric(
        small_components[
            'proposal_probability_maximum_delta_r7_minus_r5'
        ], errors='coerce'
    ).mean())
    r7_budget = budget_summary[
        budget_summary['checkpoint'] == 'r7'
    ]
    router_x3 = r7_budget[
        r7_budget['budget_policy'] == 'router_x3'
    ]
    oracle_x3 = r7_budget[
        r7_budget['budget_policy'] == 'oracle_target_x3'
    ]
    router_small_within8 = float(
        router_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    oracle_small_within8 = float(
        oracle_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    deployment_budget_gap = oracle_small_within8 - router_small_within8
    selected_epoch = int(r7_best_epoch)
    mean_dice_best_epoch = int(
        metric_best_rows['validation_mean_dice']['epoch']
    )
    checkpoint_selection_mismatch = selected_epoch != mean_dice_best_epoch

    mechanism_evidence = {
        'component_exposure_imbalance': bool(never_selected_fraction > 0.10),
        'low_resolution_component_collision': bool(
            heldout_small_collision_fraction > 0.10
            or heldout_assignment_ceiling < 0.95
        ),
        'contrastive_train_heldout_margin_divergence': bool(
            np.isfinite(fit_small_margin_delta)
            and np.isfinite(heldout_small_margin_delta)
            and fit_small_margin_delta > heldout_small_margin_delta + 0.10
        ),
        'small_proposal_signal_regressed': bool(
            small_proposal_probability_delta < 0.0
        ),
        'deployment_volume_router_limits_coverage': bool(
            deployment_budget_gap > 0.10
        ),
        'checkpoint_objective_mismatch': bool(checkpoint_selection_mismatch),
        'r7_mean_dice_regressed': bool(case_dice_delta < 0.0),
        'r7_representation_failure_extreme': bool(
            float(r7_result['best_validation_summary'][
                'small_component_representation_failure_fraction'
            ]) > 0.75
        ),
    }
    if mechanism_evidence['contrastive_train_heldout_margin_divergence']:
        dominant_mechanism = (
            'low-resolution local-contrastive over-specialization with '
            'poor held-out satellite transfer'
        )
    elif mechanism_evidence['component_exposure_imbalance']:
        dominant_mechanism = (
            'incomplete atomic-component exposure in the stochastic curriculum'
        )
    elif mechanism_evidence['low_resolution_component_collision']:
        dominant_mechanism = (
            'global proposal-grid resolution and component-collision ceiling'
        )
    else:
        dominant_mechanism = (
            'proposal supervision/refinement mismatch rather than missing '
            'clinical or arterial routing capacity'
        )
    recommended_r8 = {
        'architecture': (
            'Preserve the r5 V2.2 global/arterial/clinical path; add a bounded '
            'native-resolution local satellite-proposal head and fuse it as '
            'a zero-initialized residual, leaving the coarse global proposal '
            'responsible for burden and broad localization.'
        ),
        'curriculum': (
            'Replace stochastic inverse-size component draws with a '
            'coverage-guaranteed component cycle before repeats, stratified '
            'by subvoxel/micro/tiny/small tier and subject.'
        ),
        'objective': (
            'Apply component contrast at native local resolution, normalize '
            'per component, cap its gradient contribution against the main '
            'segmentation loss, and retain the coarse proposal loss.'
        ),
        'deployment': (
            'Train and calibrate the predicted-volume router on fold-fit data '
            'only; use predicted rather than target volume for any inference '
            'budget and report the oracle budget only as a diagnostic ceiling.'
        ),
        'selection': (
            'Use a Pareto-safe checkpoint rule that requires no mean-Dice '
            'regression and explicit held-out small-component improvement; '
            'do not optimize the present composite score alone.'
        ),
        'next_stage': (
            'Build an R8 read-only architecture/loss preflight first, then one '
            'fold-0 pilot. Five-fold and locked validation/test evaluation '
            'remain unauthorized.'
        ),
    }

    source_hashes_after = {
        str(path): file_sha256(path) for path in source_paths
    }
    final_checks = {
        **initial_checks,
        'fit_component_count_matches_frozen_872': int(
            (resolution['development_partition'] == 'fold0_fit').sum()
        ) == 872,
        'exposure_covers_every_frozen_fit_component': len(exposure)
        == len(component_df),
        'paired_case_metrics_have_20_subjects': len(paired_case) == 20,
        'paired_component_metrics_are_nonempty': len(paired_component) > 0,
        'margin_audit_has_both_checkpoints': set(margins['checkpoint'])
        == {'r5', 'r7'},
        'margin_audit_has_fit_and_heldout': set(
            margins['development_partition']
        ) == {'fold0_fit', 'fold0_heldout'},
        'all_finite_margin_inputs': bool(np.isfinite(pd.to_numeric(
            margins['positive_logit_mean'], errors='coerce'
        )).all()),
        'deployment_budget_uses_heldout_development_only': bool(
            budgets['subject_id'].isin(fold0_heldout_df['subject_id']).all()
        ),
        'development_target_access_has_exactly_102_subjects': (
            len(target_access_frame) == 102
            and target_access_frame['subject_id'].nunique() == 102
        ),
        'development_target_access_excludes_locked_splits': bool(
            (~target_access_frame['locked_validation_target_read']).all()
            and (~target_access_frame['sealed_test_target_read']).all()
        ),
        'r5_checkpoint_unchanged': r5_sha_before
        == file_sha256(R5_BEST_CHECKPOINT_PATH),
        'r7_checkpoint_unchanged': r7_sha_before
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'all_source_artifacts_unchanged': source_hashes_before
        == source_hashes_after,
        'optimizer_not_constructed': True,
        'backward_not_called': True,
        'optimizer_steps_are_zero': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    diagnostic_passed = all(final_checks.values())

    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': R7_FAILURE_DIAGNOSTIC_VERSION,
        'code_revision': R7_FAILURE_DIAGNOSTIC_CODE_REVISION,
        'r5_best_epoch': r5_best_epoch,
        'r7_best_epoch': r7_best_epoch,
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_sha256': r7_sha_before,
        'observed_effects': {
            'mean_paired_case_dice_delta_r7_minus_r5': case_dice_delta,
            'small_proposal_probability_delta_r7_minus_r5': (
                small_proposal_probability_delta
            ),
            'eligible_components_never_selected_fraction': (
                never_selected_fraction
            ),
            'heldout_small_global_voxel_collision_fraction': (
                heldout_small_collision_fraction
            ),
            'heldout_distinct_assignment_ceiling_fraction': (
                heldout_assignment_ceiling
            ),
            'fit_small_logit_margin_delta_r7_minus_r5': (
                fit_small_margin_delta
            ),
            'heldout_small_logit_margin_delta_r7_minus_r5': (
                heldout_small_margin_delta
            ),
            'r7_router_x3_small_within_8mm_fraction': router_small_within8,
            'r7_oracle_x3_small_within_8mm_fraction': oracle_small_within8,
            'deployment_to_oracle_small_within_8mm_gap': (
                deployment_budget_gap
            ),
            'selected_checkpoint_epoch': selected_epoch,
            'mean_dice_best_epoch': mean_dice_best_epoch,
        },
        'mechanism_evidence': mechanism_evidence,
        'dominant_failure_mechanism': dominant_mechanism,
        'recommended_r8_contract': recommended_r8,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': False,
        'backward_called': False,
        'optimizer_steps': 0,
        'r8_preflight_implementation_allowed': bool(diagnostic_passed),
        'r8_training_authorized': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(summary, R7_FAILURE_SUMMARY_PATH)
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(value)}
        for name, value in final_checks.items()
    ])
    atomic_write_csv(checks_frame, R7D_CHECKS_PATH)
    output_hashes = {
        path.name: file_sha256(path) for path in R7D_OUTPUT_PATHS
    }
    gate = {
        **summary,
        'diagnostic_complete': bool(diagnostic_passed),
        'diagnostic_passed': bool(diagnostic_passed),
        'checks': final_checks,
        'source_sha256_before': source_hashes_before,
        'source_sha256_after': source_hashes_after,
        'output_sha256': output_hashes,
        'diagnostic_root': str(R7_FAILURE_DIAGNOSTIC_ROOT),
        'summary_path': str(R7_FAILURE_SUMMARY_PATH),
    }
    atomic_write_json(gate, R7_FAILURE_GATE_PATH)
    if not diagnostic_passed:
        failed = [
            name for name, passed in final_checks.items() if not passed
        ]
        raise RuntimeError(
            'TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC FAILED. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nComponent exposure by tier:')
    display(exposure_summary)
    print('\nProposal-grid resolution summary:')
    display(resolution_summary)
    print('\nTrain-versus-held-out proposal-logit margins:')
    display(margin_summary)
    print('\nDeployment-legal predicted-volume budget audit:')
    display(budget_summary)
    print('\nBranch reliance comparison:')
    display(branch_comparison)
    print('\nDiagnostic checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Dominant mechanism:                 {dominant_mechanism}')
    print(f'Eligible components never selected: {never_selected_fraction:.4f}')
    print(f'Fit small-margin change r7-r5:      {fit_small_margin_delta:+.4f}')
    print(f'Heldout small-margin change r7-r5:  {heldout_small_margin_delta:+.4f}')
    print(f'R7 router-x3 small within 8 mm:     {router_small_within8:.4f}')
    print(f'R7 oracle-x3 small within 8 mm:     {oracle_small_within8:.4f}')
    print(f'Mean-Dice best / selected epoch:    {mean_dice_best_epoch} / {selected_epoch}')
    print('Optimizer / backward / steps:       0 / 0 / 0')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('R8 preflight implementation allowed: True')
    print('R8 training authorized:             False')
    print('Full five-fold training authorized: False')
    print(f'Diagnostic summary:                 {R7_FAILURE_SUMMARY_PATH}')
    print(f'Diagnostic gate:                    {R7_FAILURE_GATE_PATH}')
    print(
        '\nNext: construct the read-only R8 dual-resolution satellite-proposal '
        'and component-cycled curriculum preflight; do not train yet.'
    )


run_r7_failure_mechanism_diagnostic()






R8_MODEL_VERSION = 'trial8_vascufate_net_v2_3_dual_resolution_satellite'
R8_PREFLIGHT_VERSION = 'v1_r5_warmstart_component_cycle_native_satellite'
R8_CODE_REVISION = (
    'r1_native_component_absolute_calibration_pareto_guard'
)
R8_SEED = 20261007
R8_SATELLITE_GATE_BOUND = 0.20
R8_SATELLITE_FEATURE_CHANNELS = 16
R8_COMPONENT_POSITIVE_PATCHES = 112
R8_TOPOLOGY_NEGATIVE_PATCHES = 28
R8_BRAIN_NEGATIVE_PATCHES = 28
R8_SCHEDULE_EPOCHS = 40
R8_NATIVE_INNER_RADIUS_MM = 2.0
R8_NATIVE_OUTER_RADIUS_MM = 12.0
R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR = 1.0
R8_MAXIMUM_NEW_PARAMETERS = 250_000
R8_TIER_QUOTAS = {
    'subvoxel_or_single_voxel': 2,
    'micro': 76,
    'tiny': 27,
    'small': 7,
}

R8_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / R8_MODEL_VERSION / R8_PREFLIGHT_VERSION
)
R8_PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
R8_ARCHITECTURE_PATH = R8_PREFLIGHT_ROOT / 'r8_architecture_config.json'
R8_TRAINING_POLICY_PATH = R8_PREFLIGHT_ROOT / 'r8_training_policy.json'
R8_SCHEDULE_PATH = R8_PREFLIGHT_ROOT / 'r8_component_cycle_schedule.csv'
R8_SCHEDULE_SUMMARY_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_component_cycle_summary.csv'
)
R8_NATIVE_TARGET_AUDIT_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_native_component_target_audit.csv'
)
R8_PARAMETER_PATH = R8_PREFLIGHT_ROOT / 'r8_parameter_audit.csv'
R8_OUTPUT_PATH = R8_PREFLIGHT_ROOT / 'r8_output_shape_audit.csv'
R8_NEUTRALITY_PATH = R8_PREFLIGHT_ROOT / 'r8_neutrality_audit.csv'
R8_GRADIENT_PATH = R8_PREFLIGHT_ROOT / 'r8_gradient_audit.csv'
R8_LOSS_PATH = R8_PREFLIGHT_ROOT / 'r8_loss_smoke_audit.csv'
R8_CHECKS_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_checks.csv'
R8_GATE_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_gate.json'

R8_OUTPUT_ARTIFACTS = [
    R8_ARCHITECTURE_PATH,
    R8_TRAINING_POLICY_PATH,
    R8_SCHEDULE_PATH,
    R8_SCHEDULE_SUMMARY_PATH,
    R8_NATIVE_TARGET_AUDIT_PATH,
    R8_PARAMETER_PATH,
    R8_OUTPUT_PATH,
    R8_NEUTRALITY_PATH,
    R8_GRADIENT_PATH,
    R8_LOSS_PATH,
    R8_CHECKS_PATH,
]


def r8_existing_preflight_complete():
    if not R8_GATE_PATH.exists():
        return False
    gate = load_json(R8_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('model_version') == R8_MODEL_VERSION
        and gate.get('preflight_version') == R8_PREFLIGHT_VERSION
        and gate.get('code_revision') == R8_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_failure_diagnostic_sha256')
        == file_sha256(R7_FAILURE_GATE_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R8_OUTPUT_ARTIFACTS
    )


def r8_tier_records():
    records = {}
    for tier in R8_TIER_QUOTAS:
        frame = component_df[
            component_df['component_tier'].astype(str) == tier
        ].sort_values(['subject_id', 'atomic_component_id'])
        records[tier] = frame.to_dict('records')
        if not records[tier]:
            raise RuntimeError(f'R8 component tier {tier!r} is empty.')
    return records


R8_COMPONENT_RECORDS_BY_TIER = r8_tier_records()


def r8_tier_permutation(tier, cycle_index):
    tier_index = list(R8_TIER_QUOTAS).index(str(tier))
    rng = np.random.default_rng(
        R8_SEED + tier_index * 10_000_019 + int(cycle_index) * 1_000_003
    )
    count = len(R8_COMPONENT_RECORDS_BY_TIER[str(tier)])
    return rng.permutation(count).tolist()


def r8_component_for_position(tier, position):
    records = R8_COMPONENT_RECORDS_BY_TIER[str(tier)]
    cycle_index = int(position) // len(records)
    offset = int(position) % len(records)
    permutation = r8_tier_permutation(tier, cycle_index)
    item = dict(records[int(permutation[offset])])
    item['r8_tier_cycle_index'] = int(cycle_index)
    item['r8_tier_cycle_offset'] = int(offset)
    return item


def r8_build_epoch_schedule(epoch):
    epoch = int(epoch)
    if epoch < 1:
        raise ValueError('R8 epoch must be positive.')
    positives = []
    for tier, quota in R8_TIER_QUOTAS.items():
        for slot in range(int(quota)):
            position = (epoch - 1) * int(quota) + slot
            component = r8_component_for_position(tier, position)
            subject_id = str(component['subject_id'])
            jitter_rng = np.random.default_rng(
                R8_SEED + epoch * 10_000_019
                + list(R8_TIER_QUOTAS).index(tier) * 104_729 + slot
            )
            center = np.asarray([
                component['centroid_x_vox'],
                component['centroid_y_vox'],
                component['centroid_z_vox'],
            ], dtype=np.float64)
            jitter = np.asarray([
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-3, 4),
            ])
            shape = np.asarray(
                pilot_mask_records[subject_id]['shape'], dtype=np.int64
            )
            center = np.clip(np.rint(center).astype(np.int64) + jitter, 0, shape - 1)
            positives.append({
                'epoch': epoch,
                'sample_type': 'positive',
                'subject_id': subject_id,
                'center_native': tuple(int(value) for value in center),
                'atomic_component_id': int(
                    component['atomic_component_id']
                ),
                'component_tier': tier,
                'component_volume_ml': float(
                    component['component_volume_ml']
                ),
                'r8_tier_cycle_index': int(
                    component['r8_tier_cycle_index']
                ),
                'r8_tier_cycle_offset': int(
                    component['r8_tier_cycle_offset']
                ),
            })
    if len(positives) != R8_COMPONENT_POSITIVE_PATCHES:
        raise RuntimeError(
            f'R8 epoch {epoch} has {len(positives)} positive patches.'
        )
    inherited = pilot_build_epoch_schedule(epoch)
    negatives = [
        dict(item) for item in inherited
        if str(item['sample_type']) != 'positive'
    ]
    if sum(str(item['sample_type']) == 'topology_negative' for item in negatives) != 28:
        raise RuntimeError('R8 topology-negative count changed.')
    if sum(str(item['sample_type']) == 'brain_negative' for item in negatives) != 28:
        raise RuntimeError('R8 brain-negative count changed.')
    for item in negatives:
        item['epoch'] = epoch
        item['r8_tier_cycle_index'] = -1
        item['r8_tier_cycle_offset'] = -1
    schedule = positives + negatives
    rng = np.random.default_rng(R8_SEED + epoch * 1_299_721)
    rng.shuffle(schedule)
    if len(schedule) != 168:
        raise RuntimeError(f'R8 epoch {epoch} has {len(schedule)} samples.')
    return schedule


def r8_schedule_audit():
    schedules = []
    repeated = []
    for epoch in range(1, R8_SCHEDULE_EPOCHS + 1):
        first = r8_build_epoch_schedule(epoch)
        second = r8_build_epoch_schedule(epoch)
        if first != second:
            raise RuntimeError(f'R8 epoch {epoch} schedule is not deterministic.')
        schedules.extend(first)
        repeated.extend(second)
    frame = pd.DataFrame(schedules)
    positive = frame[frame['sample_type'].astype(str) == 'positive'].copy()
    positive['component_key'] = (
        positive['subject_id'].astype(str) + ':'
        + positive['atomic_component_id'].astype(int).astype(str)
    )
    expected_keys = set(
        component_df[
            component_df['component_tier'].astype(str).isin(R8_TIER_QUOTAS)
        ].assign(component_key=lambda value:
            value['subject_id'].astype(str) + ':'
            + value['atomic_component_id'].astype(int).astype(str)
        )['component_key']
    )
    observed_keys = set(positive['component_key'])
    rows = []
    for tier, quota in R8_TIER_QUOTAS.items():
        tier_positive = positive[
            positive['component_tier'].astype(str) == tier
        ]
        counts = tier_positive.groupby('component_key').size()
        records = R8_COMPONENT_RECORDS_BY_TIER[tier]
        completed_cycles_are_unique = True
        for cycle_index, cycle in tier_positive.groupby('r8_tier_cycle_index'):
            keys = cycle['component_key'].tolist()
            if len(cycle) == len(records) and len(keys) != len(set(keys)):
                completed_cycles_are_unique = False
        rows.append({
            'component_tier': tier,
            'tier_components': len(records),
            'quota_per_epoch': int(quota),
            'total_selections_40_epochs': int(len(tier_positive)),
            'observed_components': int(counts.size),
            'never_selected_components': int(len(records) - counts.size),
            'minimum_selections': int(counts.min()),
            'median_selections': float(counts.median()),
            'mean_selections': float(counts.mean()),
            'maximum_selections': int(counts.max()),
            'selection_gini': r7d_gini(counts.to_numpy()),
            'maximum_minus_minimum_selections': int(
                counts.max() - counts.min()
            ),
            'complete_cycles_have_no_duplicates': bool(
                completed_cycles_are_unique
            ),
        })
    summary = pd.DataFrame(rows)
    per_epoch = positive.groupby('epoch').agg(
        positive_patches=('component_key', 'size'),
        unique_components=('component_key', 'nunique'),
        unique_subjects=('subject_id', 'nunique'),
    ).reset_index()
    summary.attrs['all_eligible_components_covered'] = (
        observed_keys == expected_keys
    )
    summary.attrs['minimum_unique_positive_subjects_per_epoch'] = int(
        per_epoch['unique_subjects'].min()
    )
    atomic_write_csv(frame, R8_SCHEDULE_PATH)
    atomic_write_csv(summary, R8_SCHEDULE_SUMMARY_PATH)
    return frame, summary, per_epoch, expected_keys, observed_keys


def r8_native_component_targets(row, local_origin, component_id):
    subject_id = str(row['subject_id'])
    binary, soft, support, spacing = r7d_component_support(row)
    labels, count = ndimage.label(
        support, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    expected = len(component_records.get(subject_id, []))
    if int(count) != int(expected):
        raise RuntimeError(
            f'{subject_id}: R8 reconstructed {count} components; expected {expected}.'
        )
    component_id = int(component_id)
    if not 1 <= component_id <= count:
        raise RuntimeError(f'{subject_id}: invalid component {component_id}.')
    selected = labels == component_id
    selected_canvas, start_a = symmetric_pad(
        selected.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    support_canvas, start_b = symmetric_pad(
        support.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(support, dtype=np.uint8),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if not (start_a == start_b == start_c):
        raise RuntimeError(f'{subject_id}: R8 target padding disagreement.')
    exact = crop_at(
        selected_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    full = crop_at(
        support_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    domain = crop_at(
        domain_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    if not bool(exact.any()):
        raise RuntimeError(f'{subject_id}: R8 native exact component is empty.')
    distance = ndimage.distance_transform_edt(~exact, sampling=spacing)
    negative = (
        (distance >= R8_NATIVE_INNER_RADIUS_MM)
        & (distance <= R8_NATIVE_OUTER_RADIUS_MM)
        & domain & ~full
    )
    if not bool(negative.any()):
        raise RuntimeError(f'{subject_id}: R8 native annulus is empty.')
    retained_fraction = float(exact.sum() / max(selected.sum(), 1))
    return {
        'native_component_target': exact[None],
        'native_component_negative_annulus': negative[None],
        'spacing': spacing,
        'component_native_voxels': int(selected.sum()),
        'component_patch_voxels': int(exact.sum()),
        'component_retained_fraction': retained_fraction,
        'negative_annulus_voxels': int(negative.sum()),
        'binary': binary,
        'soft': soft,
    }


def r8_native_target_probe(schedule_frame):
    probes = []
    positive = schedule_frame[
        schedule_frame['sample_type'].astype(str) == 'positive'
    ]
    for tier in R8_TIER_QUOTAS:
        candidates = positive[
            positive['component_tier'].astype(str) == tier
        ].head(8)
        for item in candidates.to_dict('records'):
            subject_id = str(item['subject_id'])
            row = fit_row_by_subject[subject_id]
            local_origin = pilot_origin_from_center(
                item['center_native'], pilot_mask_records[subject_id]['shape']
            )
            target = r8_native_component_targets(
                row, local_origin, item['atomic_component_id']
            )
            probes.append({
                'subject_id': subject_id,
                'component_tier': tier,
                'atomic_component_id': int(item['atomic_component_id']),
                'component_native_voxels': target['component_native_voxels'],
                'component_patch_voxels': target['component_patch_voxels'],
                'component_retained_fraction': target[
                    'component_retained_fraction'
                ],
                'negative_annulus_voxels': target[
                    'negative_annulus_voxels'
                ],
                'exact_and_annulus_disjoint': not bool(np.any(
                    target['native_component_target']
                    & target['native_component_negative_annulus']
                )),
                'target_arrays_read_split': 'fold0_fit',
                'locked_validation_target_read': False,
                'test_target_read': False,
            })
    frame = pd.DataFrame(probes)
    atomic_write_csv(frame, R8_NATIVE_TARGET_AUDIT_PATH)
    return frame


class NativeSatelliteProposalHead(nn.Module):
    def __init__(self, local_channels=32, hidden=R8_SATELLITE_FEATURE_CHANNELS):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(local_channels + 2, hidden, 1, bias=False),
            nn.GroupNorm(group_count(hidden), hidden), nn.GELU(),
        )
        self.refine = ResidualBlock3D(hidden, hidden, dropout=0.05)
        self.output = nn.Conv3d(hidden, 1, 1)

    def forward(self, local_feature, coarse_proposal, arterial_local):
        feature = self.project(torch.cat([
            local_feature, torch.sigmoid(coarse_proposal), arterial_local,
        ], dim=1))
        return self.output(self.refine(feature))


class VascuFateNetR8(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.base = VascuFateNetV22(schema)
        self.satellite_head = NativeSatelliteProposalHead()
        self.satellite_gate = ConfidenceGate(
            confidence_dim=5, output_dim=1,
            bound=R8_SATELLITE_GATE_BOUND, hidden_dim=32,
        )
        self._decoder_feature = None
        self._decoder_hook = self.base.local_model.decoder1.register_forward_hook(
            self._capture_decoder_feature
        )
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def _capture_decoder_feature(self, module, inputs, output):
        self._decoder_feature = output

    def load_r5_base_state(self, state):
        self.base.load_state_dict(state, strict=True)

    def activate_satellite_gate_for_gradient_test(self, fraction=0.5):
        self.satellite_gate.activate_for_gradient_test(fraction)

    def reset_satellite_gate_to_zero(self):
        with torch.no_grad():
            self.satellite_gate.network[-1].weight.zero_()
            self.satellite_gate.network[-1].bias.zero_()

    def forward(self, **kwargs):
        self._decoder_feature = None
        output = self.base(**kwargs)
        if self._decoder_feature is None:
            raise RuntimeError('R8 decoder feature hook did not fire.')
        arterial_local = sample_global_to_local(
            output['arterial_global_map'][:, None],
            kwargs['local_origin_in_canvas'],
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        if bool(kwargs.get('force_global_dropout', False)):
            coarse = torch.zeros_like(output['global_proposal_local'])
            arterial_local = torch.zeros_like(arterial_local)
        else:
            coarse = output['global_proposal_local']
        satellite_logits = self.satellite_head(
            self._decoder_feature, coarse, arterial_local
        )
        confidence = torch.cat([
            kwargs['confidence_features'],
            torch.sigmoid(output['lesion_presence_logit'])[:, None],
        ], dim=-1)
        gate = self.satellite_gate(confidence)
        base_logits = output['logits']
        fused_logits = base_logits + gate[..., None, None, None] * satellite_logits
        self._decoder_feature = None
        return {
            **output,
            'base_logits': base_logits,
            'satellite_logits': satellite_logits,
            'satellite_gate': gate,
            'logits': fused_logits,
        }


def r8_satellite_component_loss(
    logits, exact_target, negative_annulus, component_valid,
):
    logits = logits.float()
    exact_target = exact_target.bool()
    negative_annulus = negative_annulus.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    rows = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]]
        negative = logits[batch_index][negative_annulus[batch_index]]
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError('R8 component loss received an empty region.')
        hard_count = min(
            int(negative.numel()),
            max(32, 4 * int(positive.numel())),
        )
        hard_negative = torch.topk(negative, hard_count).values
        positive_bce = F.softplus(-positive).mean()
        negative_bce = F.softplus(hard_negative).mean()
        rank = F.relu(
            1.0 - positive.mean() + hard_negative.mean()
        )
        absolute_floor = F.softplus(
            R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR - positive.mean()
        )
        loss = (
            0.35 * positive_bce + 0.30 * negative_bce
            + 0.20 * rank + 0.15 * absolute_floor
        )
        losses.append(loss)
        rows.append({
            'positive_voxels': int(positive.numel()),
            'negative_voxels': int(negative.numel()),
            'hard_negative_voxels': int(hard_count),
            'positive_logit_mean': float(positive.mean().detach().cpu()),
            'hard_negative_logit_mean': float(
                hard_negative.mean().detach().cpu()
            ),
            'relative_logit_margin': float(
                (positive.mean() - hard_negative.mean()).detach().cpu()
            ),
            'positive_bce': float(positive_bce.detach().cpu()),
            'hard_negative_bce': float(negative_bce.detach().cpu()),
            'rank_loss': float(rank.detach().cpu()),
            'absolute_positive_floor_loss': float(
                absolute_floor.detach().cpu()
            ),
            'total_satellite_loss': float(loss.detach().cpu()),
        })
    if not losses:
        return logits.sum() * 0.0, []
    return torch.stack(losses).mean(), rows


def r8_model_input_batch(sample, device):
    return {
        key: torch.from_numpy(np.asarray(sample[key])[None]).to(device)
        for key in MODEL_INPUT_KEYS_V22
    }


def r8_gradient_norm(module):
    values = [
        parameter.grad.detach().float().norm().cpu().item()
        for parameter in module.parameters()
        if parameter.grad is not None
    ]
    if not values:
        return 0.0, False, True
    finite = bool(np.isfinite(values).all())
    return float(np.linalg.norm(values)), True, finite


def run_r8_dual_resolution_preflight():
    if r8_existing_preflight_complete():
        gate = load_json(R8_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Total parameters: {gate["total_parameters"]:,}')
        print(f'Preflight gate:   {R8_GATE_PATH}')
        print('No source checkpoint or preflight artifact was rewritten.')
        return

    r7_failure_gate = load_json(R7_FAILURE_GATE_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r5_recomputed = dict(
        r7_result['r5_recomputed_under_r7_metric_contract']
    )
    r5_mean_dice = float(r5_recomputed['positive_mean_dice'])
    r5_global_dice = float(r5_recomputed['positive_global_dice'])
    r8_mean_dice_eligibility_floor = r5_mean_dice - 0.005
    r8_global_dice_eligibility_floor = r5_global_dice - 0.010
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_failure_sha = file_sha256(R7_FAILURE_GATE_PATH)
    upstream_checks = {
        'r7_failure_diagnostic_complete': (
            r7_failure_gate.get('diagnostic_complete') is True
        ),
        'r7_failure_diagnostic_passed': (
            r7_failure_gate.get('diagnostic_passed') is True
        ),
        'r8_preflight_allowed_by_diagnostic': (
            r7_failure_gate.get('r8_preflight_implementation_allowed') is True
        ),
        'r8_training_was_locked_by_diagnostic': (
            r7_failure_gate.get('r8_training_authorized') is False
        ),
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r5_recomputed_baseline_is_finite': bool(np.isfinite([
            r5_mean_dice, r5_global_dice,
        ]).all()),
        'r5_checkpoint_hash_matches': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'fold0_fit_count_is_82': len(fold0_fit_df) == 82,
        'fold0_heldout_targets_not_used': len(fold0_heldout_df) == 20,
        'locked_validation_targets_not_used': len(locked_validation_df) == 23,
        'sealed_test_targets_not_used': len(sealed_test_df) == 23,
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError('R8 upstream checks failed: ' + ', '.join(failed))

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT STARTING')
    print('=' * 80)
    print('Warm-start source:                  completed r5 best checkpoint')
    print('R7 checkpoint used for weights:     False')
    print('Target access:                      fold-0 fit probes only')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer / training steps:         0 / 0')

    schedule, schedule_summary, per_epoch, expected_keys, observed_keys = (
        r8_schedule_audit()
    )
    target_probe = r8_native_target_probe(schedule)

    positive_schedule = schedule[
        schedule['sample_type'].astype(str) == 'positive'
    ]
    smoke_candidates = positive_schedule[
        positive_schedule['component_tier'].astype(str).isin(['micro', 'tiny'])
    ]
    smoke_entry = None
    for item in smoke_candidates.to_dict('records'):
        if as_bool(fit_row_by_subject[str(item['subject_id'])][
            'trial8_atlas_enabled'
        ]):
            smoke_entry = item
            break
    if smoke_entry is None:
        raise RuntimeError('R8 found no atlas-enabled micro/tiny smoke case.')
    subject_id = str(smoke_entry['subject_id'])
    row = fit_row_by_subject[subject_id]
    local_origin = pilot_origin_from_center(
        smoke_entry['center_native'], pilot_mask_records[subject_id]['shape']
    )
    dataset = Trial8V22Fold0Dataset()
    dataset.epoch = int(smoke_entry['epoch'])
    dataset.schedule = [smoke_entry]
    sample = dataset[0]
    native_target = r8_native_component_targets(
        row, local_origin, smoke_entry['atomic_component_id']
    )

    if not torch.cuda.is_available():
        raise RuntimeError('R8 preflight requires an L4-class CUDA runtime.')
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < 20.0:
        raise RuntimeError(
            f'R8 preflight expected >=20 GB GPU, observed {gpu_capacity_gb:.2f} GB.'
        )
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'r5_checkpoint_pilot_version_matches': (
            checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'r5_checkpoint_code_revision_matches': (
            checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
        ),
        'r5_checkpoint_parameter_count_matches': int(
            checkpoint.get('parameter_count', -1)
        ) == EXPECTED_V22_PARAMETERS,
        'r5_checkpoint_contains_ema_state': 'ema_model_state' in checkpoint,
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 warm start: ' + ', '.join(failed))
    batch = r8_model_input_batch(sample, device)
    force_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }

    reference = VascuFateNetV22(pilot_schema).to(device).eval()
    reference.load_state_dict(checkpoint['ema_model_state'], strict=True)
    for parameter in reference.parameters():
        parameter.requires_grad_(False)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        reference_output = reference(**batch, **force_flags)
    reference_cpu = {
        key: value.detach().float().cpu()
        for key, value in reference_output.items()
        if torch.is_tensor(value)
    }
    del reference, reference_output
    gc.collect()
    torch.cuda.empty_cache()

    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(checkpoint['ema_model_state'])
    del checkpoint
    model.eval()
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    base_parameters = sum(parameter.numel() for parameter in model.base.parameters())
    satellite_head_parameters = sum(
        parameter.numel() for parameter in model.satellite_head.parameters()
    )
    satellite_gate_parameters = sum(
        parameter.numel() for parameter in model.satellite_gate.parameters()
    )
    new_parameters = satellite_head_parameters + satellite_gate_parameters
    parameter_frame = pd.DataFrame([
        {'component': 'preserved_r5_v2_2_base', 'parameters': base_parameters},
        {'component': 'native_satellite_head', 'parameters': satellite_head_parameters},
        {'component': 'bounded_satellite_gate', 'parameters': satellite_gate_parameters},
        {'component': 'new_r8_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
    ])
    atomic_write_csv(parameter_frame, R8_PARAMETER_PATH)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        neutral_output = model(**batch, **force_flags)
    peak_eval_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    preserved_keys = [
        'logits', 'auxiliary_half_logits', 'auxiliary_quarter_logits',
        'territory_node_logits', 'global_graph_context',
        'global_proposal_logits', 'lesion_presence_logit',
        'log_volume_prediction',
    ]
    neutrality_rows = []
    for key in preserved_keys:
        difference = float(torch.max(torch.abs(
            neutral_output[key].detach().float().cpu() - reference_cpu[key]
        )))
        neutrality_rows.append({
            'output': key,
            'maximum_absolute_difference_from_r5': difference,
            'exactly_preserved_with_zero_gate': difference <= 1e-6,
        })
    neutrality_frame = pd.DataFrame(neutrality_rows)
    atomic_write_csv(neutrality_frame, R8_NEUTRALITY_PATH)

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'base_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_gate': (1, 1),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
    }
    output_rows = []
    for key, expected in expected_shapes.items():
        observed = tuple(neutral_output[key].shape)
        output_rows.append({
            'output': key,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(neutral_output[key]).all().item()),
        })
    output_frame = pd.DataFrame(output_rows)
    atomic_write_csv(output_frame, R8_OUTPUT_PATH)

    zero_gate = float(neutral_output['satellite_gate'].abs().max().cpu())
    satellite_std = float(
        neutral_output['satellite_logits'].float().std().cpu()
    )
    model.activate_satellite_gate_for_gradient_test(0.5)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        activated_output = model(**batch, **force_flags)
    activated_difference = float(torch.max(torch.abs(
        activated_output['logits'].float()
        - activated_output['base_logits'].float()
    )).cpu())
    model.reset_satellite_gate_to_zero()
    del activated_output

    model.train()
    model.zero_grad(set_to_none=True)
    exact_target = torch.from_numpy(
        native_target['native_component_target'][None]
    ).to(device)
    negative_target = torch.from_numpy(
        native_target['native_component_negative_annulus'][None]
    ).to(device)
    component_valid = torch.ones(1, dtype=torch.bool, device=device)
    local_binary = torch.from_numpy(
        np.asarray(sample['target_binary'])[None]
    ).to(device)
    global_target = torch.from_numpy(
        np.asarray(sample['global_target'])[None]
    ).to(device)
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        gradient_output = model(**batch, **force_flags)
        main_loss = (
            0.60 * pilot_soft_dice_loss(
                gradient_output['logits'], local_binary
            ).mean()
            + 0.40 * pilot_focal_bce_loss(
                gradient_output['logits'], local_binary
            ).mean()
        )
        coarse_loss = (
            0.70 * pilot_soft_dice_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
            + 0.30 * pilot_focal_bce_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
        )
        satellite_loss, satellite_loss_rows = r8_satellite_component_loss(
            gradient_output['satellite_logits'], exact_target,
            negative_target, component_valid,
        )
        smoke_loss = main_loss + 0.20 * coarse_loss + 0.10 * satellite_loss
    smoke_loss.backward()
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_groups = {
        'preserved_local_segmentation_head': (
            model.base.local_model.segmentation_head
        ),
        'preserved_global_proposal_stream': model.base.global_stream,
        'native_satellite_head': model.satellite_head,
        'bounded_satellite_gate': model.satellite_gate,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_groups.items():
        norm, observed, finite = r8_gradient_norm(module)
        gradient_rows.append({
            'component': name,
            'gradient_norm': norm,
            'gradient_observed': observed,
            'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_frame = pd.DataFrame(gradient_rows)
    atomic_write_csv(gradient_frame, R8_GRADIENT_PATH)
    loss_frame = pd.DataFrame([{
        'subject_id': subject_id,
        'component_tier': str(smoke_entry['component_tier']),
        'component_id': int(smoke_entry['atomic_component_id']),
        'main_loss': float(main_loss.detach().float().cpu()),
        'coarse_whole_mask_loss': float(coarse_loss.detach().float().cpu()),
        'native_satellite_loss': float(
            satellite_loss.detach().float().cpu()
        ),
        'total_smoke_loss': float(smoke_loss.detach().float().cpu()),
        **satellite_loss_rows[0],
    }])
    atomic_write_csv(loss_frame, R8_LOSS_PATH)
    model.zero_grad(set_to_none=True)

    architecture = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'warm_start': {
            'source': 'completed_r5_best_ema',
            'checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'checkpoint_sha256': r5_sha_before,
            'r7_weights_used': False,
            'zero_gate_output_equivalence_required': True,
        },
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'native_satellite_head': {
            'input': 'native-resolution_r5_decoder_feature_plus_coarse_proposal_plus_arterial_map',
            'input_spatial_shape': list(LOCAL_PATCH_SIZE),
            'hidden_channels': R8_SATELLITE_FEATURE_CHANNELS,
            'output': 'native_resolution_satellite_logits',
            'direct_graph_to_logit_residual': False,
        },
        'fusion': {
            'type': 'zero_initialized_bounded_scalar_residual',
            'bound': R8_SATELLITE_GATE_BOUND,
            'gate_context': 'registration_confidence_plus_lesion_presence',
            'initial_final_output': 'exact_r5_logits',
        },
        'coarse_global_role': (
            'whole_brain_burden_and_broad_localization_only; '
            'no_r7_coarse_component_contrastive_loss'
        ),
        'clinical_and_arterial_role': (
            'preserved_bounded_v2_2_context; no treatment/outcome leakage'
        ),
    }
    training_policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'experiment': 'R8_fold0_pilot_only_after_separate_training_preflight',
        'component_cycle': {
            'positive_patches_per_epoch': R8_COMPONENT_POSITIVE_PATCHES,
            'tier_quotas': R8_TIER_QUOTAS,
            'rule': 'within_each_tier_every_component_once_before_repeat',
            'schedule_epochs_audited': R8_SCHEDULE_EPOCHS,
        },
        'optimization_stages': [
            {
                'epochs': '1-4',
                'trainable': ['native_satellite_head'],
                'frozen': ['r5_base', 'satellite_gate'],
                'purpose': 'learn_absolute_native_component_evidence_before_fusion',
            },
            {
                'epochs': '5-12',
                'trainable': ['native_satellite_head', 'satellite_gate'],
                'frozen': ['r5_base'],
                'purpose': 'learn_bounded_residual_without_erasing_r5_behavior',
            },
            {
                'epochs': '13+',
                'trainable': ['all'],
                'learning_rate_policy': (
                    'r5_base_2e-5_satellite_2e-4_gate_1e-4'
                ),
                'purpose': 'low_rate_joint_calibration',
            },
        ],
        'loss_contract': {
            'main_segmentation': 'preserved_r5_dice_plus_focal',
            'coarse_proposal': 'whole_mask_only',
            'native_satellite': (
                'per_component_positive_bce_plus_hard_negative_bce_plus_'
                'ranking_plus_absolute_positive_logit_floor'
            ),
            'coarse_component_contrastive': False,
            'volume_case_weights': False,
            'topk_ohem': False,
        },
        'checkpoint_eligibility': {
            'r5_recomputed_mean_dice': r5_mean_dice,
            'r5_recomputed_global_dice': r5_global_dice,
            'mean_dice_minimum': r8_mean_dice_eligibility_floor,
            'global_dice_minimum': r8_global_dice_eligibility_floor,
            'small_component_improvement_required': True,
            'deployment_metrics_only': True,
            'oracle_volume_budget_used_for_selection': False,
            'selection_order': [
                'eligibility_guards', 'mean_dice', 'small_case_dice',
                'small_component_recall', 'lesion_f1',
            ],
        },
        'fold0_heldout_targets_for_training': False,
        'locked_validation_targets_for_training': False,
        'test_targets_for_training': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(architecture, R8_ARCHITECTURE_PATH)
    atomic_write_json(training_policy, R8_TRAINING_POLICY_PATH)

    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    preflight_checks = {
        **upstream_checks,
        **checkpoint_checks,
        'tier_quotas_sum_to_112': sum(R8_TIER_QUOTAS.values()) == 112,
        'schedule_has_40_epochs': schedule['epoch'].nunique() == 40,
        'every_epoch_has_168_samples': bool(
            (schedule.groupby('epoch').size() == 168).all()
        ),
        'every_epoch_has_112_positive_samples': bool(
            (schedule[schedule['sample_type'] == 'positive']
             .groupby('epoch').size() == 112).all()
        ),
        'all_eligible_components_are_covered': observed_keys == expected_keys,
        'no_eligible_component_is_never_selected': bool(
            (schedule_summary['never_selected_components'] == 0).all()
        ),
        'component_selection_counts_differ_by_at_most_one': bool(
            (schedule_summary['maximum_minus_minimum_selections'] <= 1).all()
        ),
        'complete_tier_cycles_have_no_duplicates': bool(
            schedule_summary['complete_cycles_have_no_duplicates'].all()
        ),
        'component_cycle_gini_below_0p10': bool(
            (schedule_summary['selection_gini'] < 0.10).all()
        ),
        'native_target_probes_cover_four_tiers': (
            target_probe['component_tier'].nunique() == 4
        ),
        'all_native_target_probes_are_nonempty': bool(
            (target_probe['component_patch_voxels'] > 0).all()
        ),
        'all_native_annuli_are_nonempty': bool(
            (target_probe['negative_annulus_voxels'] > 0).all()
        ),
        'all_native_targets_and_annuli_are_disjoint': bool(
            target_probe['exact_and_annulus_disjoint'].all()
        ),
        'all_native_target_probes_retain_at_least_95_percent': bool(
            (target_probe['component_retained_fraction'] >= 0.95).all()
        ),
        'preserved_base_parameter_count_matches_r5': (
            base_parameters == EXPECTED_V22_PARAMETERS
        ),
        'new_parameter_count_within_limit': (
            0 < new_parameters <= R8_MAXIMUM_NEW_PARAMETERS
        ),
        'all_zero_gate_outputs_match_r5': bool(
            neutrality_frame['exactly_preserved_with_zero_gate'].all()
        ),
        'satellite_gate_is_exactly_zero_initially': zero_gate == 0.0,
        'satellite_logits_are_nonconstant': satellite_std > 1e-6,
        'activated_satellite_gate_changes_final_logits': (
            activated_difference > 1e-7
        ),
        'all_output_shapes_match': bool(output_frame['shape_matches'].all()),
        'all_outputs_are_finite': bool(output_frame['finite'].all()),
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss).item()),
        'satellite_loss_contains_absolute_positive_floor': (
            float(loss_frame['absolute_positive_floor_loss'].iloc[0]) > 0.0
        ),
        **gradient_checks,
        'peak_eval_memory_below_20gb': peak_eval_memory_gb < 20.0,
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'r7_checkpoint_not_loaded_for_weights': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in preflight_checks.items()
    ])
    atomic_write_csv(checks_frame, R8_CHECKS_PATH)
    preflight_passed = all(preflight_checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R8_OUTPUT_ARTIFACTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_used_for_weights': False,
        'r7_failure_diagnostic_path': str(R7_FAILURE_GATE_PATH),
        'r7_failure_diagnostic_sha256': r7_failure_sha,
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'gpu': gpu_name,
        'gpu_capacity_gb': gpu_capacity_gb,
        'peak_eval_memory_gb': peak_eval_memory_gb,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'schedule_components': len(expected_keys),
        'minimum_unique_positive_subjects_per_epoch': int(
            per_epoch['unique_subjects'].min()
        ),
        'native_target_probes': len(target_probe),
        'optimizer_steps': 0,
        'training_performed': False,
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'r8_pilot_implementation_allowed': bool(preflight_passed),
        'r8_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'checks': preflight_checks,
        'artifact_sha256': artifact_hashes,
        'architecture_path': str(R8_ARCHITECTURE_PATH),
        'training_policy_path': str(R8_TRAINING_POLICY_PATH),
    }
    atomic_write_json(gate, R8_GATE_PATH)
    del model, neutral_output, gradient_output, batch
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in preflight_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR8 component-cycle summary:')
    display(schedule_summary)
    print('\nR8 parameter count:')
    display(parameter_frame)
    print('\nR8 output shapes:')
    display(output_frame)
    print('\nR8 gradient smoke:')
    display(gradient_frame)
    print('\nR8 preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Warm-start checkpoint:              r5 epoch {r5_result["best_epoch"]}')
    print('R7 weights used:                    False')
    print(f'Preserved base parameters:          {base_parameters:,}')
    print(f'New satellite parameters:           {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Eligible components cycled:         {len(expected_keys)}')
    print(f'Minimum unique subjects per epoch:  {int(per_epoch["unique_subjects"].min())}')
    print(f'Peak gradient memory GB:            {peak_gradient_memory_gb:.3f}')
    print('Coarse component contrastive loss:  disabled')
    print('Native absolute calibration loss:   enabled')
    print('Initial final logits equal r5:      True')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Training performed:                 False')
    print('R8 pilot implementation allowed:    True')
    print('R8 pilot training authorized:       False')
    print('Full five-fold training authorized: False')
    print(f'Architecture:                       {R8_ARCHITECTURE_PATH}')
    print(f'Training policy:                    {R8_TRAINING_POLICY_PATH}')
    print(f'Preflight gate:                     {R8_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 R8 pilot loader/loss/training '
        'preflight from this frozen contract; do not train yet.'
    )


run_r8_dual_resolution_preflight()












R8P_PREFLIGHT_VERSION = 'v1_stagewise_native_satellite_pilot_contract'
R8P_CODE_REVISION = 'r1_negative_hard_background_pareto_resume_guard'
R8P_PILOT_VERSION = 'v2_3_fold0_dual_resolution_satellite_r8'
R8P_PILOT_CODE_REVISION = 'r1_r5_warmstart_stagewise_native_satellite'
R8P_SEED = 20261008
R8P_MAX_EPOCHS = 40
R8P_MINIMUM_EPOCHS = 20
R8P_EARLY_STOPPING_PATIENCE = 10
R8P_EARLY_STOPPING_MINIMUM_DELTA = 0.002
R8P_ACCUMULATION_STEPS = 4
R8P_HEAD_LR = 2.0e-4
R8P_GATE_LR = 1.0e-4
R8P_BASE_LR = 2.0e-5
R8P_MINIMUM_LR_FRACTION = 0.10
R8P_WEIGHT_DECAY = 1.0e-4
R8P_GRADIENT_CLIP_NORM = 2.0
R8P_SATELLITE_LOSS_COEFFICIENT = 0.10
R8P_NEGATIVE_BACKGROUND_WEIGHT = 0.25
R8P_NEGATIVE_BACKGROUND_TOPK = 4096
R8P_VALIDATION_OVERLAP = 0.50
R8P_LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
R8P_PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
R8P_EXPECTED_PARAMETERS = 16_885_385

R8P_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / R8_MODEL_VERSION / R8P_PREFLIGHT_VERSION
)
R8P_ROOT.mkdir(parents=True, exist_ok=True)
R8P_POLICY_PATH = R8P_ROOT / 'r8_fold0_pilot_policy.json'
R8P_RESUME_PATH = R8P_ROOT / 'r8_resume_contract.json'
R8P_LOADER_AUDIT_PATH = R8P_ROOT / 'r8_loader_case_audit.csv'
R8P_BATCH_AUDIT_PATH = R8P_ROOT / 'r8_batch_tensor_audit.csv'
R8P_STAGE_AUDIT_PATH = R8P_ROOT / 'r8_stage_trainability_audit.csv'
R8P_LR_AUDIT_PATH = R8P_ROOT / 'r8_learning_rate_audit.csv'
R8P_OPTIMIZER_AUDIT_PATH = R8P_ROOT / 'r8_optimizer_group_audit.csv'
R8P_LOSS_AUDIT_PATH = R8P_ROOT / 'r8_loss_smoke_audit.csv'
R8P_GRADIENT_AUDIT_PATH = R8P_ROOT / 'r8_stage_gradient_audit.csv'
R8P_CACHE_AUDIT_PATH = R8P_ROOT / 'r8_cached_inference_audit.csv'
R8P_SELECTION_AUDIT_PATH = R8P_ROOT / 'r8_checkpoint_policy_audit.csv'
R8P_CHECKS_PATH = R8P_ROOT / 'r8_fold0_pilot_preflight_checks.csv'
R8P_GATE_PATH = R8P_ROOT / 'r8_fold0_pilot_preflight_gate.json'

R8P_OUTPUT_ARTIFACTS = [
    R8P_POLICY_PATH, R8P_RESUME_PATH, R8P_LOADER_AUDIT_PATH,
    R8P_BATCH_AUDIT_PATH, R8P_STAGE_AUDIT_PATH, R8P_LR_AUDIT_PATH,
    R8P_OPTIMIZER_AUDIT_PATH, R8P_LOSS_AUDIT_PATH,
    R8P_GRADIENT_AUDIT_PATH, R8P_CACHE_AUDIT_PATH,
    R8P_SELECTION_AUDIT_PATH, R8P_CHECKS_PATH,
]

R8P_PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / R8_MODEL_VERSION / R8P_PILOT_VERSION
)
R8P_PILOT_CONFIG_PATH = R8P_PILOT_ROOT / 'pilot_config.json'
R8P_PILOT_START_GATE_PATH = R8P_PILOT_ROOT / 'pilot_start_gate.json'
R8P_LAST_CHECKPOINT_PATH = R8P_PILOT_ROOT / 'last_checkpoint.pt'
R8P_BEST_CHECKPOINT_PATH = R8P_PILOT_ROOT / 'best_checkpoint.pt'
R8P_HISTORY_PATH = R8P_PILOT_ROOT / 'training_history.csv'
R8P_RESULT_GATE_PATH = R8P_PILOT_ROOT / 'pilot_result_gate.json'


def r8p_existing_preflight_complete():
    if not R8P_GATE_PATH.exists():
        return False
    gate = load_json(R8P_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('preflight_version') == R8P_PREFLIGHT_VERSION
        and gate.get('code_revision') == R8P_CODE_REVISION
        and gate.get('r8_architecture_gate_sha256') == file_sha256(R8_GATE_PATH)
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R8P_OUTPUT_ARTIFACTS
    )


def r8p_parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in sorted(model.named_parameters()):
        digest.update(name.encode('utf-8'))
        digest.update(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        )
    return digest.hexdigest()


def r8p_stage(epoch):
    epoch = int(epoch)
    if not 1 <= epoch <= R8P_MAX_EPOCHS:
        raise ValueError(f'R8 pilot epoch {epoch} is outside 1..40.')
    if epoch <= 4:
        return 'satellite_head_only'
    if epoch <= 12:
        return 'satellite_head_and_gate'
    return 'low_rate_joint_calibration'


def r8p_branch_lr(epoch, start_epoch, peak_lr):
    epoch = int(epoch)
    start_epoch = int(start_epoch)
    if epoch < start_epoch:
        return 0.0
    if epoch <= start_epoch + 1:
        return float(peak_lr * (epoch - start_epoch + 1) / 2.0)
    minimum = peak_lr * R8P_MINIMUM_LR_FRACTION
    denominator = max(R8P_MAX_EPOCHS - (start_epoch + 1), 1)
    progress = np.clip(
        (epoch - (start_epoch + 1)) / denominator, 0.0, 1.0
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * float(progress)))
    return float(minimum + (peak_lr - minimum) * cosine)


def r8p_learning_rates(epoch):
    return {
        'r5_base': r8p_branch_lr(epoch, 13, R8P_BASE_LR),
        'native_satellite_head': r8p_branch_lr(epoch, 1, R8P_HEAD_LR),
        'bounded_satellite_gate': r8p_branch_lr(epoch, 5, R8P_GATE_LR),
    }


def r8p_set_requires_grad(module, value):
    for parameter in module.parameters():
        parameter.requires_grad_(bool(value))


def r8p_apply_stage(model, epoch, optimizer=None):
    stage = r8p_stage(epoch)
    model.train()
    base_trainable = stage == 'low_rate_joint_calibration'
    gate_trainable = stage != 'satellite_head_only'
    r8p_set_requires_grad(model.base, base_trainable)
    r8p_set_requires_grad(model.satellite_head, True)
    r8p_set_requires_grad(model.satellite_gate, gate_trainable)
    if not base_trainable:
        model.base.eval()
    if not gate_trainable:
        model.satellite_gate.eval()
    learning_rates = r8p_learning_rates(epoch)
    if optimizer is not None:
        observed_names = set()
        for group in optimizer.param_groups:
            name = str(group['group_name'])
            observed_names.add(name)
            group['lr'] = float(learning_rates[name])
        if observed_names != set(learning_rates):
            raise RuntimeError('R8 optimizer groups do not match LR policy.')
    return stage, learning_rates


def r8p_optimizer_groups(model):
    groups = [
        {
            'group_name': 'r5_base',
            'params': list(model.base.parameters()),
            'lr': 0.0,
            'weight_decay': R8P_WEIGHT_DECAY,
        },
        {
            'group_name': 'native_satellite_head',
            'params': list(model.satellite_head.parameters()),
            'lr': R8P_HEAD_LR / 2.0,
            'weight_decay': R8P_WEIGHT_DECAY,
        },
        {
            'group_name': 'bounded_satellite_gate',
            'params': list(model.satellite_gate.parameters()),
            'lr': 0.0,
            'weight_decay': 0.0,
        },
    ]
    identifiers = [
        id(parameter) for group in groups for parameter in group['params']
    ]
    if len(identifiers) != len(set(identifiers)):
        raise RuntimeError('R8 optimizer parameter groups overlap.')
    model_identifiers = {id(parameter) for parameter in model.parameters()}
    if set(identifiers) != model_identifiers:
        raise RuntimeError('R8 optimizer groups do not cover the model exactly.')
    return groups


class Trial8R8Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = r8_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = dict(self.schedule[int(index)])
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        positive = str(entry['sample_type']) == 'positive'
        if positive:
            native = r8_native_component_targets(
                row, local_origin, entry['atomic_component_id']
            )
            if not bool((targets['target_soft'] > 0).any()):
                raise RuntimeError(
                    f'{subject_id}: R8 positive patch lost soft support.'
                )
        else:
            if bool((targets['target_soft'] > 0).any()):
                raise RuntimeError(
                    f'{subject_id}: R8 negative patch contains target.'
                )
            native = {
                'native_component_target': np.zeros(
                    (1, *LOCAL_PATCH_SIZE), dtype=bool
                ),
                'native_component_negative_annulus': np.zeros(
                    (1, *LOCAL_PATCH_SIZE), dtype=bool
                ),
                'component_patch_voxels': 0,
                'negative_annulus_voxels': 0,
                'component_retained_fraction': 1.0,
            }
        rng = np.random.default_rng(
            R8P_SEED + self.epoch * 10_000_019 + int(index) * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'native_component_target': native['native_component_target'],
            'native_component_negative_annulus': native[
                'native_component_negative_annulus'
            ],
            'native_component_valid': np.asarray(positive, dtype=bool),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
            'component_patch_voxels': np.asarray(
                native['component_patch_voxels'], dtype=np.int64
            ),
            'negative_annulus_voxels': np.asarray(
                native['negative_annulus_voxels'], dtype=np.int64
            ),
            'component_retained_fraction': np.asarray(
                native['component_retained_fraction'], dtype=np.float32
            ),
        })
        return sample


def r8p_negative_background_loss(logits, component_valid):
    valid = component_valid.bool().reshape(-1)
    losses = []
    selected_counts = []
    for batch_index in range(logits.shape[0]):
        if bool(valid[batch_index]):
            continue
        flattened = logits[batch_index].float().reshape(-1)
        hard_count = min(R8P_NEGATIVE_BACKGROUND_TOPK, flattened.numel())
        hard = torch.topk(flattened, k=hard_count, largest=True).values
        losses.append(F.softplus(hard).mean())
        selected_counts.append(hard_count)
    if not losses:
        return logits.float().sum() * 0.0, 0
    return torch.stack(losses).mean(), int(sum(selected_counts))


def r8p_whole_proposal_loss(logits, target):
    dice = pilot_soft_dice_loss(logits, target).mean()
    focal = pilot_focal_bce_loss(logits, target).mean()
    return 0.65 * dice + 0.35 * focal, dice, focal


def r8p_compute_training_loss(
    model, cpu_batch, device, epoch, dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    if dropout_flags_override is None:
        rng = np.random.default_rng(R8P_SEED + int(epoch) * 1_000_003)
        flags = pilot_dropout_flags(rng)
    else:
        flags = {name: bool(value) for name, value in dropout_flags_override.items()}
    required_flags = {
        'force_atlas_dropout', 'force_vascular_dropout',
        'force_occlusion_dropout', 'force_clinical_dropout',
        'force_global_dropout', 'force_arterial_dropout',
    }
    if set(flags) != required_flags:
        raise RuntimeError('R8 dropout flags do not match the frozen schema.')
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool()
        & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    proposal_loss, proposal_dice, proposal_focal = r8p_whole_proposal_loss(
        output['global_proposal_logits'], batch['global_target'].float()
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].float().sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], batch['global_target'] > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'],
        global_territory_target, arterial_valid,
    )
    satellite_positive, satellite_rows = r8_satellite_component_loss(
        output['satellite_logits'], batch['native_component_target'],
        batch['native_component_negative_annulus'],
        batch['native_component_valid'],
    )
    satellite_negative, hard_background_voxels = (
        r8p_negative_background_loss(
            output['satellite_logits'], batch['native_component_valid']
        )
    )
    satellite_objective = (
        satellite_positive
        + R8P_NEGATIVE_BACKGROUND_WEIGHT * satellite_negative
    )
    preserved_objective = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    total = (
        preserved_objective
        + R8P_SATELLITE_LOSS_COEFFICIENT * satellite_objective
    )
    parts = {
        'total_loss': total,
        'preserved_objective': preserved_objective,
        'main_loss': main_loss,
        **main_parts,
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'proposal_whole_mask_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_focal,
        'presence_loss': presence_loss,
        'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'satellite_positive_loss': satellite_positive,
        'satellite_negative_background_loss': satellite_negative,
        'satellite_objective': satellite_objective,
        'satellite_loss_coefficient': torch.as_tensor(
            R8P_SATELLITE_LOSS_COEFFICIENT,
            device=total.device, dtype=torch.float32,
        ),
        'hard_background_voxels': torch.as_tensor(
            float(hard_background_voxels),
            device=total.device, dtype=torch.float32,
        ),
        'native_components_in_batch': batch[
            'native_component_valid'
        ].float().sum(),
    }
    return total, parts, satellite_rows, output


def r8p_cached_local_from_global(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    model._decoder_feature = None
    base_output = pilot_local_from_cache(
        model.base, batch, cache,
        force_global_dropout=force_global_dropout,
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    if model._decoder_feature is None:
        raise RuntimeError('R8 cached path did not capture decoder feature.')
    coarse = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    if bool(force_global_dropout):
        coarse = torch.zeros_like(coarse)
        arterial_local = torch.zeros_like(arterial_local)
    satellite_logits = model.satellite_head(
        model._decoder_feature, coarse, arterial_local
    )
    confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    gate = model.satellite_gate(confidence)
    base_logits = base_output['logits']
    result = {
        **base_output,
        'base_logits': base_logits,
        'satellite_logits': satellite_logits,
        'satellite_gate': gate,
        'logits': base_logits + gate[..., None, None, None] * satellite_logits,
        'arterial_global_map': cache['arterial_map'],
        'global_proposal_local': coarse,
    }
    model._decoder_feature = None
    return result


def r8p_r5_small_component_recall(r5_baseline):
    rows = r5_baseline.get('component_tier_metrics', [])
    selected = [
        row for row in rows
        if str(row.get('component_tier')) in set(COMPONENT_TIERS[:4])
    ]
    denominator = sum(int(row.get('components', 0)) for row in selected)
    if denominator <= 0:
        raise RuntimeError('R5 baseline has no small-component audit rows.')
    numerator = sum(
        int(row.get('components', 0))
        * float(row.get('final_component_recall', 0.0))
        for row in selected
    )
    return float(numerator / denominator)


def r8p_r5_small_case_dice(r5_summary):
    bins = dict(r5_summary.get('lesion_bin_mean_dice', {}))
    values = [
        float(bins[name]) for name in ('tiny', 'small')
        if name in bins and np.isfinite(float(bins[name]))
    ]
    if not values:
        raise RuntimeError('R5 baseline has no finite tiny/small case Dice.')
    return float(np.mean(values))


def r8p_checkpoint_evaluation(metrics, baseline):
    required = [
        'positive_mean_dice', 'positive_global_dice',
        'small_case_mean_dice', 'small_component_recall', 'lesion_f1',
    ]
    if any(name not in metrics for name in required):
        raise RuntimeError('R8 checkpoint metrics are incomplete.')
    if not all(np.isfinite(float(metrics[name])) for name in required):
        raise RuntimeError('R8 checkpoint metrics are non-finite.')
    guards = {
        'mean_dice_floor': float(metrics['positive_mean_dice'])
        >= float(baseline['mean_dice_floor']),
        'global_dice_floor': float(metrics['positive_global_dice'])
        >= float(baseline['global_dice_floor']),
        'small_component_improved': float(metrics['small_component_recall'])
        > float(baseline['small_component_recall']),
    }
    key = (
        float(metrics['positive_mean_dice']),
        float(metrics['small_case_mean_dice']),
        float(metrics['small_component_recall']),
        float(metrics['lesion_f1']),
        float(metrics['positive_global_dice']),
    )
    return all(guards.values()), guards, key


def run_r8p_fold0_pilot_preflight():
    if r8p_existing_preflight_complete():
        gate = load_json(R8P_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 FOLD-0 PILOT PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Pilot version: {gate["pilot_version"]}')
        print(f'Preflight gate: {R8P_GATE_PATH}')
        print('No source checkpoint, model checkpoint, or artifact was rewritten.')
        return

    r8_gate = load_json(R8_GATE_PATH)
    r8_policy = load_json(R8_TRAINING_POLICY_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_baseline = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
    r5_summary = dict(r5_baseline['validation_summary'])
    r5_small_component_recall = r8p_r5_small_component_recall(r5_baseline)
    r5_small_case_dice = r8p_r5_small_case_dice(r5_summary)
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r8_sha = file_sha256(R8_GATE_PATH)
    upstream_artifacts_match = all(
        (R8_PREFLIGHT_ROOT / name).exists()
        and file_sha256(R8_PREFLIGHT_ROOT / name) == expected
        for name, expected in r8_gate.get('artifact_sha256', {}).items()
    )
    upstream_checks = {
        'r8_architecture_preflight_passed': (
            r8_gate.get('preflight_passed') is True
        ),
        'r8_all_architecture_checks_passed': (
            len(r8_gate.get('checks', {})) > 0
            and all(bool(value) for value in r8_gate['checks'].values())
        ),
        'r8_preflight_version_matches': (
            r8_gate.get('preflight_version') == R8_PREFLIGHT_VERSION
        ),
        'r8_code_revision_matches': (
            r8_gate.get('code_revision') == R8_CODE_REVISION
        ),
        'r8_artifact_hashes_match': upstream_artifacts_match,
        'r8_pilot_implementation_allowed': (
            r8_gate.get('r8_pilot_implementation_allowed') is True
        ),
        'r8_pilot_was_training_locked': (
            r8_gate.get('r8_pilot_training_authorized') is False
        ),
        'full_fivefold_was_locked': (
            r8_gate.get('full_fivefold_training_authorized') is False
        ),
        'r8_parameter_count_matches': (
            int(r8_gate.get('total_parameters', -1))
            == R8P_EXPECTED_PARAMETERS
        ),
        'r8_warm_start_is_r5': (
            str(r8_gate.get('r5_checkpoint_path'))
            == str(R5_BEST_CHECKPOINT_PATH)
        ),
        'r8_warm_start_hash_matches': (
            r8_gate.get('r5_checkpoint_sha256') == r5_sha_before
        ),
        'r7_weights_remain_excluded': (
            r8_gate.get('r7_checkpoint_used_for_weights') is False
        ),
        'r5_result_complete': r5_result.get('pilot_complete') is True,
        'r5_baseline_metrics_finite': bool(np.isfinite([
            float(r5_summary['positive_mean_dice']),
            float(r5_summary['positive_global_dice']),
            r5_small_case_dice, r5_small_component_recall,
        ]).all()),
        'fold0_partition_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_and_test_counts_match': (
            len(locked_validation_df) == 23 and len(sealed_test_df) == 23
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 PILOT PREFLIGHT UPSTREAM FAILURE. Do not continue. '
            'Failed: ' + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R8 FOLD-0 PILOT LOADER/LOSS/TRAINING PREFLIGHT STARTING')
    print('=' * 80)
    print('Warm start:                         r5 best EMA (epoch 23)')
    print('R7 weights used:                    False')
    print('Target access:                      fold-0 fit smoke cases only')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer steps:                    0')

    dataset = Trial8R8Fold0Dataset()
    selected_indices = []
    for tier in R8_TIER_QUOTAS:
        selected_indices.append(next(
            index for index, entry in enumerate(dataset.schedule)
            if str(entry['sample_type']) == 'positive'
            and str(entry['component_tier']) == tier
        ))
    selected_indices.extend([
        next(index for index, entry in enumerate(dataset.schedule)
             if str(entry['sample_type']) == 'topology_negative'),
        next(index for index, entry in enumerate(dataset.schedule)
             if str(entry['sample_type']) == 'brain_negative'),
    ])
    loader_rows = []
    positive_sample = None
    negative_sample = None
    for index in selected_indices:
        sample = dataset[index]
        float_arrays = [
            np.asarray(value) for value in sample.values()
            if isinstance(value, np.ndarray)
            and np.issubdtype(np.asarray(value).dtype, np.floating)
        ]
        row = {
            'subject_id': sample['subject_id'],
            'sample_type': sample['sample_type'],
            'component_tier': sample['component_tier'],
            'atomic_component_id': int(sample['atomic_component_id']),
            'target_soft_support_voxels': int(
                (sample['target_soft'] > 0).sum()
            ),
            'native_component_voxels': int(
                sample['native_component_target'].sum()
            ),
            'native_negative_annulus_voxels': int(
                sample['native_component_negative_annulus'].sum()
            ),
            'native_component_valid': bool(
                sample['native_component_valid']
            ),
            'component_retained_fraction': float(
                sample['component_retained_fraction']
            ),
            'all_float_arrays_finite': all(
                np.isfinite(value).all() for value in float_arrays
            ),
            'target_arrays_read_split': 'fold0_fit',
            'fold0_heldout_target_read': False,
            'locked_validation_target_read': False,
            'test_target_read': False,
        }
        loader_rows.append(row)
        if (
            positive_sample is None
            and sample['sample_type'] == 'positive'
            and sample['component_tier'] == 'micro'
        ):
            positive_sample = sample
        if negative_sample is None and sample['sample_type'] == 'brain_negative':
            negative_sample = sample
    if positive_sample is None or negative_sample is None:
        raise RuntimeError('R8 smoke samples were not resolved.')
    loader_frame = pd.DataFrame(loader_rows)
    atomic_write_csv(loader_frame, R8P_LOADER_AUDIT_PATH)

    positive_batch = torch.utils.data.default_collate([positive_sample])
    negative_batch = torch.utils.data.default_collate([negative_sample])
    expected_shapes = {
        'image': (1, 15, *LOCAL_PATCH_SIZE),
        'global_image': (1, 15, *GLOBAL_INPUT_SHAPE),
        'target_binary': (1, 1, *LOCAL_PATCH_SIZE),
        'target_soft': (1, 1, *LOCAL_PATCH_SIZE),
        'global_target': (1, 1, *GLOBAL_INPUT_SHAPE),
        'native_component_target': (1, 1, *LOCAL_PATCH_SIZE),
        'native_component_negative_annulus': (1, 1, *LOCAL_PATCH_SIZE),
        'local_origin_in_canvas': (1, 3),
    }
    batch_rows = []
    for name, expected in expected_shapes.items():
        observed = tuple(positive_batch[name].shape)
        batch_rows.append({
            'tensor': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(
                positive_batch[name].float()
            ).all().item()),
        })
    batch_frame = pd.DataFrame(batch_rows)
    atomic_write_csv(batch_frame, R8P_BATCH_AUDIT_PATH)

    random.seed(R8P_SEED)
    np.random.seed(R8P_SEED)
    torch.manual_seed(R8P_SEED)
    torch.cuda.manual_seed_all(R8P_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda':
        raise RuntimeError('Trial8 R8 fold-0 pilot preflight requires CUDA.')
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(checkpoint['ema_model_state'])
    del checkpoint
    total_parameters = sum(
        parameter.numel() for parameter in model.parameters()
    )
    parameter_digest_before = r8p_parameter_digest(model)
    optimizer_groups = r8p_optimizer_groups(model)
    r8p_apply_stage(model, 1)
    optimizer = torch.optim.AdamW(
        optimizer_groups, betas=(0.9, 0.99), eps=1e-8
    )
    optimizer_rows = []
    for group in optimizer.param_groups:
        optimizer_rows.append({
            'group_name': str(group['group_name']),
            'parameters': int(sum(
                parameter.numel() for parameter in group['params']
            )),
            'initial_lr': float(group['lr']),
            'weight_decay': float(group['weight_decay']),
            'optimizer_state_entries': int(sum(
                parameter in optimizer.state for parameter in group['params']
            )),
        })
    optimizer_frame = pd.DataFrame(optimizer_rows)
    atomic_write_csv(optimizer_frame, R8P_OPTIMIZER_AUDIT_PATH)

    stage_rows = []
    for epoch in (1, 4, 5, 12, 13, 40):
        stage, rates = r8p_apply_stage(model, epoch, optimizer)
        stage_rows.append({
            'epoch': epoch,
            'stage': stage,
            'trainable_parameters': int(sum(
                parameter.numel() for parameter in model.parameters()
                if parameter.requires_grad
            )),
            'base_trainable_parameters': int(sum(
                parameter.numel() for parameter in model.base.parameters()
                if parameter.requires_grad
            )),
            'satellite_head_trainable_parameters': int(sum(
                parameter.numel()
                for parameter in model.satellite_head.parameters()
                if parameter.requires_grad
            )),
            'satellite_gate_trainable_parameters': int(sum(
                parameter.numel()
                for parameter in model.satellite_gate.parameters()
                if parameter.requires_grad
            )),
            **{f'{name}_lr': value for name, value in rates.items()},
        })
    stage_frame = pd.DataFrame(stage_rows)
    atomic_write_csv(stage_frame, R8P_STAGE_AUDIT_PATH)
    lr_frame = stage_frame[[
        'epoch', 'stage', 'r5_base_lr', 'native_satellite_head_lr',
        'bounded_satellite_gate_lr',
    ]].copy()
    atomic_write_csv(lr_frame, R8P_LR_AUDIT_PATH)

    no_dropout = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    gradient_rows = []
    loss_rows = []
    gradient_probes = [
        (1, 'positive', positive_batch),
        (1, 'brain_negative', negative_batch),
        (5, 'positive', positive_batch),
        (13, 'positive', positive_batch),
    ]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    for epoch, sample_role, cpu_batch in gradient_probes:
        stage, rates = r8p_apply_stage(model, epoch, optimizer)
        model.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type='cuda', dtype=torch.bfloat16, enabled=True
        ):
            loss, parts, satellite_rows, output = r8p_compute_training_loss(
                model, cpu_batch, device, epoch,
                dropout_flags_override=no_dropout,
            )
        if not bool(torch.isfinite(loss).item()):
            raise RuntimeError('R8 stage loss smoke is non-finite.')
        loss.backward()
        group_modules = {
            'r5_base': model.base,
            'native_satellite_head': model.satellite_head,
            'bounded_satellite_gate': model.satellite_gate,
        }
        for group_name, module in group_modules.items():
            norm, observed, finite = r8_gradient_norm(module)
            gradient_rows.append({
                'epoch': epoch,
                'stage': stage,
                'sample_role': sample_role,
                'parameter_group': group_name,
                'gradient_norm': norm,
                'gradient_observed': observed,
                'gradient_finite': finite,
                'group_lr': float(rates[group_name]),
            })
        loss_record = {
            'epoch': epoch,
            'stage': stage,
            'sample_role': sample_role,
        }
        for name, value in parts.items():
            loss_record[name] = float(value.detach().float().cpu())
        if satellite_rows:
            loss_record.update({
                f'native_{name}': value
                for name, value in satellite_rows[0].items()
            })
        loss_rows.append(loss_record)
        del output, loss, parts
        model.zero_grad(set_to_none=True)
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_frame = pd.DataFrame(gradient_rows)
    loss_frame = pd.DataFrame(loss_rows)
    atomic_write_csv(gradient_frame, R8P_GRADIENT_AUDIT_PATH)
    atomic_write_csv(loss_frame, R8P_LOSS_AUDIT_PATH)

    r8p_apply_stage(model, 1, optimizer)
    model.eval()
    positive_device = pilot_to_device(positive_batch, device)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        direct = model(
            **{key: positive_device[key] for key in MODEL_INPUT_KEYS_V22},
            **no_dropout,
        )
        cache = pilot_global_cache(model.base, positive_device)
        cached = r8p_cached_local_from_global(
            model, positive_device, cache,
        )
    cache_rows = []
    for name in ('logits', 'base_logits', 'satellite_logits'):
        difference = float(torch.max(torch.abs(
            direct[name].float() - cached[name].float()
        )).cpu())
        cache_rows.append({
            'output': name,
            'maximum_absolute_direct_vs_cached_difference': difference,
            'within_bfloat16_tolerance': difference <= 0.02,
        })
    initial_gate_abs_max = float(
        direct['satellite_gate'].float().abs().max().cpu()
    )
    initial_final_base_difference = float(torch.max(torch.abs(
        direct['logits'].float() - direct['base_logits'].float()
    )).cpu())
    cache_frame = pd.DataFrame(cache_rows)
    atomic_write_csv(cache_frame, R8P_CACHE_AUDIT_PATH)
    del direct, cached, cache, positive_device

    baseline = {
        'r5_mean_dice': float(r5_summary['positive_mean_dice']),
        'r5_global_dice': float(r5_summary['positive_global_dice']),
        'mean_dice_floor': float(
            r8_policy['checkpoint_eligibility']['mean_dice_minimum']
        ),
        'global_dice_floor': float(
            r8_policy['checkpoint_eligibility']['global_dice_minimum']
        ),
        'small_case_mean_dice': r5_small_case_dice,
        'small_component_recall': r5_small_component_recall,
    }
    passing_metrics = {
        'positive_mean_dice': baseline['mean_dice_floor'] + 0.010,
        'positive_global_dice': baseline['global_dice_floor'] + 0.010,
        'small_case_mean_dice': baseline['small_case_mean_dice'] + 0.010,
        'small_component_recall': baseline['small_component_recall'] + 0.010,
        'lesion_f1': 0.20,
        'oracle_volume_budgeted_recall': 0.99,
    }
    mean_failure = dict(passing_metrics)
    mean_failure['positive_mean_dice'] = baseline['mean_dice_floor'] - 0.001
    small_failure = dict(passing_metrics)
    small_failure['small_component_recall'] = baseline[
        'small_component_recall'
    ]
    oracle_variant = dict(passing_metrics)
    oracle_variant['oracle_volume_budgeted_recall'] = 0.0
    selection_tests = []
    for name, metrics in [
        ('eligible_improvement', passing_metrics),
        ('mean_dice_regression', mean_failure),
        ('no_small_component_improvement', small_failure),
        ('oracle_metric_changed_only', oracle_variant),
    ]:
        eligible, guards, key = r8p_checkpoint_evaluation(metrics, baseline)
        selection_tests.append({
            'scenario': name,
            'eligible': eligible,
            'selection_key': str(key),
            **guards,
        })
    selection_frame = pd.DataFrame(selection_tests)
    atomic_write_csv(selection_frame, R8P_SELECTION_AUDIT_PATH)

    parameter_digest_after = r8p_parameter_digest(model)
    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    expected_trainable = {
        1: 14_481,
        4: 14_481,
        5: 14_706,
        12: 14_706,
        13: R8P_EXPECTED_PARAMETERS,
        40: R8P_EXPECTED_PARAMETERS,
    }

    def gradient_entry(epoch, role, group):
        selected = gradient_frame[
            (gradient_frame['epoch'] == epoch)
            & (gradient_frame['sample_role'] == role)
            & (gradient_frame['parameter_group'] == group)
        ]
        if len(selected) != 1:
            raise RuntimeError('R8 gradient audit row is not unique.')
        return selected.iloc[0]

    policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'warm_start': {
            'source': 'r5_best_ema_epoch_23',
            'checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'checkpoint_sha256': r5_sha_before,
            'r7_weights_used': False,
            'parent_checkpoint_migration_allowed': False,
        },
        'epoch_schedule': {
            'maximum_epochs': R8P_MAX_EPOCHS,
            'minimum_epochs': R8P_MINIMUM_EPOCHS,
            'early_stopping_patience': R8P_EARLY_STOPPING_PATIENCE,
            'minimum_delta_for_patience_only': (
                R8P_EARLY_STOPPING_MINIMUM_DELTA
            ),
            'checkpoint_save_rule': 'every_strict_lexicographic_improvement',
            'metrics_printed_every_epoch': True,
        },
        'optimization_stages': r8_policy['optimization_stages'],
        'learning_rates': {
            'r5_base_peak': R8P_BASE_LR,
            'satellite_head_peak': R8P_HEAD_LR,
            'satellite_gate_peak': R8P_GATE_LR,
            'branch_local_two_epoch_warmup': True,
            'cosine_decay_to_fraction': R8P_MINIMUM_LR_FRACTION,
            'accumulation_steps': R8P_ACCUMULATION_STEPS,
            'gradient_clip_norm': R8P_GRADIENT_CLIP_NORM,
        },
        'loss': {
            'preserved_r5_main_and_auxiliary_objective': True,
            'coarse_proposal_whole_mask_only': True,
            'coarse_component_contrastive_disabled': True,
            'native_component_objective': (
                'positive_bce_plus_hard_negative_bce_plus_ranking_plus_'
                'absolute_positive_floor'
            ),
            'target_free_negative_objective': (
                'top4096_native_satellite_hard_background_softplus'
            ),
            'negative_background_weight': R8P_NEGATIVE_BACKGROUND_WEIGHT,
            'satellite_loss_coefficient': R8P_SATELLITE_LOSS_COEFFICIENT,
            'volume_case_weights': False,
            'topk_ohem_on_main_segmentation': False,
        },
        'validation': {
            'partition': 'fold0_heldout_development_only',
            'frequency': 'every_epoch',
            'local_threshold_grid': R8P_LOCAL_THRESHOLDS.tolist(),
            'proposal_threshold_grid': R8P_PROPOSAL_THRESHOLDS.tolist(),
            'sliding_window_overlap': R8P_VALIDATION_OVERLAP,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'per_epoch_diagnostics': [
                'mean_median_global_dice', 'lesion_precision_recall_f1',
                'lesion_volume_bin_dice', 'all_and_small_component_recall',
                'satellite_positive_and_hard_negative_logit_margin',
                'proposal_precision_ap_and_volume_ratio',
                'gate_distribution_and_thresholds',
            ],
            'every_five_epoch_diagnostics': [
                'satellite_ablation', 'global_ablation',
                'arterial_ablation', 'clinical_ablation',
                'r5_base_parameter_drift',
            ],
        },
        'checkpoint_eligibility': {
            **baseline,
            'small_component_improvement_required': True,
            'selection_order': [
                'positive_mean_dice', 'small_case_mean_dice',
                'small_component_recall', 'lesion_f1',
                'positive_global_dice',
            ],
            'oracle_metrics_used_for_selection': False,
            'thresholds_saved_with_checkpoint': True,
            'if_no_r8_epoch_is_eligible': (
                'declare_directional_failure_and_retain_r5_best_as_fallback'
            ),
        },
        'output_paths': {
            'pilot_root': str(R8P_PILOT_ROOT),
            'best_checkpoint': str(R8P_BEST_CHECKPOINT_PATH),
            'last_checkpoint': str(R8P_LAST_CHECKPOINT_PATH),
            'history': str(R8P_HISTORY_PATH),
            'result_gate': str(R8P_RESULT_GATE_PATH),
        },
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(policy, R8P_POLICY_PATH)
    policy_sha256 = file_sha256(R8P_POLICY_PATH)
    resume_contract = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'pilot_policy_sha256': policy_sha256,
        'accepted_resume_source': 'same_r8_pilot_last_checkpoint_only',
        'fresh_start_source': 'r5_best_ema_plus_deterministic_r8_initialization',
        'r7_checkpoint_migration_allowed': False,
        'r5_checkpoint_migration_after_epoch_zero_allowed': False,
        'deployment_fallback_if_no_eligible_r8_checkpoint': str(
            R5_BEST_CHECKPOINT_PATH
        ),
        'required_checkpoint_fields': [
            'pilot_version', 'code_revision', 'pilot_policy_sha256',
            'epoch', 'model_state', 'ema_model_state', 'optimizer_state',
            'rng_state', 'best_selection_key', 'best_thresholds',
            'optimizer_steps',
        ],
        'rng_restore_policy': (
            'normalize_cpu_and_cuda_rng_tensors_to_contiguous_cpu_uint8'
        ),
        'checkpoint_write_policy': (
            'atomic_last_each_epoch_and_atomic_best_on_every_strict_eligible_'
            'lexicographic_improvement'
        ),
        'retroactive_best_reconstruction': False,
        'mismatched_config_action': 'refuse_resume_without_overwrite',
    }
    atomic_write_json(resume_contract, R8P_RESUME_PATH)

    stage_counts_match = all(
        int(row['trainable_parameters']) == expected_trainable[int(row['epoch'])]
        for row in stage_rows
    )
    positive_loss_row = loss_frame[
        (loss_frame['epoch'] == 1)
        & (loss_frame['sample_role'] == 'positive')
    ].iloc[0]
    negative_loss_row = loss_frame[
        (loss_frame['epoch'] == 1)
        & (loss_frame['sample_role'] == 'brain_negative')
    ].iloc[0]
    oracle_a = selection_frame[
        selection_frame['scenario'] == 'eligible_improvement'
    ].iloc[0]
    oracle_b = selection_frame[
        selection_frame['scenario'] == 'oracle_metric_changed_only'
    ].iloc[0]
    checks = {
        **upstream_checks,
        'dataset_has_168_samples': len(dataset) == 168,
        'dataset_has_112_positive_samples': sum(
            str(item['sample_type']) == 'positive'
            for item in dataset.schedule
        ) == 112,
        'dataset_has_28_topology_negatives': sum(
            str(item['sample_type']) == 'topology_negative'
            for item in dataset.schedule
        ) == 28,
        'dataset_has_28_brain_negatives': sum(
            str(item['sample_type']) == 'brain_negative'
            for item in dataset.schedule
        ) == 28,
        'loader_smoke_covers_all_four_component_tiers': (
            set(loader_frame[
                loader_frame['sample_type'] == 'positive'
            ]['component_tier']) == set(R8_TIER_QUOTAS)
        ),
        'loader_smoke_covers_both_negative_roles': (
            {'topology_negative', 'brain_negative'}.issubset(
                set(loader_frame['sample_type'])
            )
        ),
        'positive_loader_cases_have_native_targets': bool((
            loader_frame[loader_frame['sample_type'] == 'positive'][
                'native_component_voxels'
            ] > 0
        ).all()),
        'positive_loader_cases_have_native_annuli': bool((
            loader_frame[loader_frame['sample_type'] == 'positive'][
                'native_negative_annulus_voxels'
            ] > 0
        ).all()),
        'negative_loader_cases_are_target_free': bool((
            loader_frame[loader_frame['sample_type'] != 'positive'][
                'target_soft_support_voxels'
            ] == 0
        ).all()),
        'all_loader_float_arrays_are_finite': bool(
            loader_frame['all_float_arrays_finite'].all()
        ),
        'all_batch_shapes_match': bool(batch_frame['shape_matches'].all()),
        'all_batch_tensors_are_finite': bool(batch_frame['finite'].all()),
        'total_parameter_count_matches': (
            total_parameters == R8P_EXPECTED_PARAMETERS
        ),
        'optimizer_groups_cover_model_exactly': (
            int(optimizer_frame['parameters'].sum()) == total_parameters
        ),
        'optimizer_groups_are_disjoint': True,
        'optimizer_state_is_empty': len(optimizer.state) == 0,
        'optimizer_steps_are_zero': True,
        'stage_trainable_counts_match': stage_counts_match,
        'base_lr_is_zero_before_epoch13': bool((
            lr_frame[lr_frame['epoch'] < 13]['r5_base_lr'] == 0.0
        ).all()),
        'gate_lr_is_zero_before_epoch5': bool((
            lr_frame[lr_frame['epoch'] < 5][
                'bounded_satellite_gate_lr'
            ] == 0.0
        ).all()),
        'head_lr_is_positive_from_epoch1': bool((
            lr_frame['native_satellite_head_lr'] > 0.0
        ).all()),
        'all_loss_smokes_are_finite': bool(np.isfinite(
            loss_frame[[
                'total_loss', 'preserved_objective', 'main_loss',
                'proposal_whole_mask_loss', 'presence_loss',
                'volume_loss', 'arterial_loss',
                'satellite_positive_loss',
                'satellite_negative_background_loss',
                'satellite_objective', 'satellite_loss_coefficient',
                'hard_background_voxels', 'native_components_in_batch',
            ]].to_numpy(dtype=np.float64)
        ).all()),
        'positive_native_absolute_floor_is_active': (
            float(positive_loss_row['native_absolute_positive_floor_loss'])
            > 0.0
        ),
        'negative_native_background_loss_is_active': (
            float(negative_loss_row['satellite_negative_background_loss'])
            > 0.0
            and float(negative_loss_row['hard_background_voxels'])
            == R8P_NEGATIVE_BACKGROUND_TOPK
        ),
        'coarse_component_contrastive_is_absent': (
            'local_contrastive_loss' not in loss_frame.columns
        ),
        'stage1_positive_head_gradient_present': bool(
            gradient_entry(1, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage1_negative_head_gradient_present': bool(
            gradient_entry(1, 'brain_negative', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage1_base_gradient_absent': not bool(
            gradient_entry(1, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage1_gate_gradient_absent': not bool(
            gradient_entry(1, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'stage2_head_gradient_present': bool(
            gradient_entry(5, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage2_gate_gradient_present': bool(
            gradient_entry(5, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'stage2_base_gradient_absent': not bool(
            gradient_entry(5, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage3_base_gradient_present': bool(
            gradient_entry(13, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage3_head_gradient_present': bool(
            gradient_entry(13, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage3_gate_gradient_present': bool(
            gradient_entry(13, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'all_observed_gradients_are_finite': bool(
            gradient_frame['gradient_finite'].all()
        ),
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'cached_validation_adapter_matches_direct_forward': bool(
            cache_frame['within_bfloat16_tolerance'].all()
        ),
        'actual_pilot_model_gate_is_zero_at_initialization': (
            initial_gate_abs_max == 0.0
        ),
        'actual_pilot_model_initial_output_equals_r5_base': (
            initial_final_base_difference == 0.0
        ),
        'eligible_checkpoint_scenario_passes': bool(
            selection_frame[
                selection_frame['scenario'] == 'eligible_improvement'
            ]['eligible'].iloc[0]
        ),
        'mean_dice_regression_is_rejected': not bool(
            selection_frame[
                selection_frame['scenario'] == 'mean_dice_regression'
            ]['eligible'].iloc[0]
        ),
        'missing_small_component_improvement_is_rejected': not bool(
            selection_frame[
                selection_frame['scenario']
                == 'no_small_component_improvement'
            ]['eligible'].iloc[0]
        ),
        'oracle_metric_cannot_change_selection': (
            bool(oracle_a['eligible']) == bool(oracle_b['eligible'])
            and str(oracle_a['selection_key']) == str(oracle_b['selection_key'])
        ),
        'checkpoint_paths_are_isolated_from_r5_and_r7': (
            R8P_BEST_CHECKPOINT_PATH not in {
                R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH
            }
            and R8P_LAST_CHECKPOINT_PATH != PILOT_LAST_CHECKPOINT_PATH
        ),
        'resume_contract_hashes_exact_pilot_policy': (
            resume_contract['pilot_policy_sha256']
            == file_sha256(R8P_POLICY_PATH)
        ),
        'no_r8_checkpoint_was_written': (
            not R8P_BEST_CHECKPOINT_PATH.exists()
            and not R8P_LAST_CHECKPOINT_PATH.exists()
        ),
        'model_parameters_unchanged_by_backward_smokes': (
            parameter_digest_before == parameter_digest_after
        ),
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'training_not_performed': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in checks.items()
    ])
    atomic_write_csv(checks_frame, R8P_CHECKS_PATH)
    preflight_passed = all(checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R8P_OUTPUT_ARTIFACTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8P_PREFLIGHT_VERSION,
        'code_revision': R8P_CODE_REVISION,
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r8_architecture_gate_path': str(R8_GATE_PATH),
        'r8_architecture_gate_sha256': r8_sha,
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_weights_used': False,
        'total_parameters': total_parameters,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'fit_targets_read': len(set(loader_frame['subject_id'])),
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': True,
        'optimizer_steps': 0,
        'training_performed': False,
        'r8_fold0_pilot_training_authorized': bool(preflight_passed),
        'full_fivefold_training_authorized': False,
        'pilot_policy_path': str(R8P_POLICY_PATH),
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'resume_contract_path': str(R8P_RESUME_PATH),
        'checks': checks,
        'artifact_sha256': artifact_hashes,
    }
    atomic_write_json(gate, R8P_GATE_PATH)

    del model, optimizer, positive_batch, negative_batch
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 FOLD-0 PILOT PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR8 loader smoke cases:')
    display(loader_frame)
    print('\nR8 staged trainability and learning rates:')
    display(stage_frame)
    print('\nR8 real-patch loss smoke:')
    display(loss_frame)
    print('\nR8 staged gradient audit:')
    display(gradient_frame)
    print('\nR8 checkpoint-policy tests:')
    display(selection_frame)
    print('\nR8 fold-0 pilot preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R8 FOLD-0 PILOT LOADER/LOSS/TRAINING PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Pilot version:                       {R8P_PILOT_VERSION}')
    print(f'Fit / held-out subjects:             {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Total parameters:                    {total_parameters:,}')
    print(f'Peak gradient memory GB:             {peak_gradient_memory_gb:.3f}')
    print('Stage 1 negative patches train head: True')
    print('Coarse component contrastive loss:   disabled')
    print('Native absolute calibration loss:    enabled')
    print('R5 mean/global Pareto floors:         enforced')
    print('Oracle metrics select checkpoints:   False')
    print('Metrics printed:                      every epoch')
    print('Fold-0 held-out targets read:         False')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Optimizer steps:                      0')
    print('Training performed:                   False')
    print('R8 fold-0 pilot training authorized:  True')
    print('Full five-fold training authorized:   False')
    print(f'Pilot policy:                         {R8P_POLICY_PATH}')
    print(f'Resume contract:                      {R8P_RESUME_PATH}')
    print(f'Preflight gate:                       {R8P_GATE_PATH}')
    print(
        '\nNext: build and run the isolated fold-0 R8 pilot training cell '
        'from this frozen gate; do not launch five folds.'
    )


run_r8p_fold0_pilot_preflight()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = (
    'r7_delayed_local_contrastive_volume_bounded_r1'
)
PILOT_VERSION = 'v2_2_fold0_local_contrastive_proposal_r7'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

R7_PREFLIGHT_VERSION = 'v1_delayed_small_component_local_contrastive'
R7_PREFLIGHT_CODE_REVISION = (
    'r1_precision_bounded_diffuse_shortcut_guard'
)
R7_COMPONENT_LOSS_MAX_COEFFICIENT = 0.10
R7_COMPONENT_LOSS_START_EPOCH = 8
R7_COMPONENT_LOSS_FULL_EPOCH = 12
R7_POSITIVE_WEIGHT = 0.40
R7_HARD_NEGATIVE_WEIGHT = 0.40
R7_RANK_WEIGHT = 0.20
R7_INNER_RADIUS_MM = 4.0
R7_OUTER_RADIUS_MM = 12.0
R7_HARD_NEGATIVE_RATIO = 4
R7_HARD_NEGATIVE_MINIMUM = 16
R7_LOGIT_MARGIN = 1.0
R7_VOLUME_BUDGET_MULTIPLIER = 3.0
R7_SMALL_COMPONENT_TIERS = set(COMPONENT_TIERS[:4])
R5_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
R5_PILOT_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
R7_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip((0.8, 0.8, 2.0), GLOBAL_DOWNSAMPLE)
)

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'
R7_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r7_local_contrastive_component_proposal' / R7_PREFLIGHT_VERSION
)
R7_PREFLIGHT_GATE_PATH = (
    R7_PREFLIGHT_ROOT / 'r7_local_contrastive_preflight_gate.json'
)
R7_POLICY_PATH = R7_PREFLIGHT_ROOT / 'r7_local_contrastive_policy.json'
R5_PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / R5_PILOT_VERSION
R5_RESULT_GATE_PATH = R5_PILOT_ROOT / 'pilot_result_gate.json'
R5_BEST_CHECKPOINT_PATH = R5_PILOT_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'
R7_R5_BASELINE_DIAGNOSTIC_PATH = (
    PILOT_DIAGNOSTIC_ROOT / 'r5_volume_bounded_baseline.json'
)
R7_FAILURE_DIAGNOSTIC_VERSION = 'v1_r5_r7_failure_mechanism_audit'
R7_FAILURE_DIAGNOSTIC_CODE_REVISION = (
    'r1_exposure_resolution_margin_volume_branch_checkpoint'
)
R7_FAILURE_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'model_diagnostics'
    / MODEL_VERSION / R7_FAILURE_DIAGNOSTIC_VERSION
)
R7_FAILURE_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
R7_FAILURE_GATE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_diagnostic_gate.json'
)
R7_FAILURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_summary.json'
)



PILOT_TARGET_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'fold0_target_access_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_schedule_audit.csv'
)
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH, R7_PREFLIGHT_GATE_PATH, R7_POLICY_PATH,
    R5_RESULT_GATE_PATH, R5_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
r7_preflight_gate = load_json(R7_PREFLIGHT_GATE_PATH)
r7_policy = load_json(R7_POLICY_PATH)
r5_result_gate = load_json(R5_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r7_component_preflight_passed': (
        r7_preflight_gate.get('preflight_passed') is True
    ),
    'r7_component_preflight_all_checks_true': (
        len(r7_preflight_gate.get('checks', {})) > 0
        and all(bool(value) for value in r7_preflight_gate['checks'].values())
    ),
    'r7_preflight_version_matches': (
        r7_preflight_gate.get('preflight_version') == R7_PREFLIGHT_VERSION
    ),
    'r7_preflight_revision_matches': (
        r7_preflight_gate.get('code_revision')
        == R7_PREFLIGHT_CODE_REVISION
    ),
    'r7_implementation_allowed': (
        r7_preflight_gate.get('r7_pilot_implementation_allowed') is True
    ),
    'r7_training_was_locked_before_this_block': (
        r7_preflight_gate.get('r7_pilot_training_authorized') is False
    ),
    'r7_architecture_parameter_count_unchanged': (
        int(r7_preflight_gate.get('architecture_parameters', -1))
        == EXPECTED_V22_PARAMETERS
        and int(r7_preflight_gate.get('new_trainable_parameters', -1)) == 0
    ),
    'r7_preflight_target_scope_is_fit_only': (
        r7_preflight_gate.get('target_arrays_read_splits') == ['fold0_fit']
        and r7_preflight_gate.get('fold0_heldout_targets_read') is False
        and r7_preflight_gate.get('locked_validation_targets_read') is False
        and r7_preflight_gate.get('test_targets_read') is False
    ),
    'r7_policy_path_matches_gate': (
        str(r7_preflight_gate.get('policy_path')) == str(R7_POLICY_PATH)
    ),
    'r7_policy_local_contrastive_contract_matches': (
        r7_policy.get('selected_component_objective', {}).get(
            'eligible_tiers'
        ) == sorted(R7_SMALL_COMPONENT_TIERS)
        and int(r7_policy.get('selected_component_objective', {}).get(
            'start_epoch', -1
        )) == R7_COMPONENT_LOSS_START_EPOCH
        and int(r7_policy.get('selected_component_objective', {}).get(
            'full_coefficient_epoch', -1
        )) == R7_COMPONENT_LOSS_FULL_EPOCH
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('maximum_coefficient', float('nan'))),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('inner_radius_mm', float('nan'))),
            R7_INNER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('outer_radius_mm', float('nan'))),
            R7_OUTER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('positive_weight', float('nan'))),
            R7_POSITIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_weight', float('nan'))),
            R7_HARD_NEGATIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('rank_weight', float('nan'))),
            R7_RANK_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_ratio', -1)) == R7_HARD_NEGATIVE_RATIO
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_minimum', -1)) == R7_HARD_NEGATIVE_MINIMUM
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('logit_margin', float('nan'))),
            R7_LOGIT_MARGIN, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get('experiment_seed', -1)) == PILOT_SEED
    ),
    'r7_policy_requires_volume_bounded_diagnostics': (
        r7_policy.get('proposal_diagnostics', {}).get(
            'raw_distance_metrics_are_not_success_gates'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_proposal_precision'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_predicted_to_target_volume_ratio'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_voxel_average_precision'
        ) is True
        and math.isclose(float(r7_policy.get(
            'proposal_diagnostics', {}
        ).get('volume_budget_multiplier', float('nan'))),
            R7_VOLUME_BUDGET_MULTIPLIER, rel_tol=0.0, abs_tol=1e-12)
    ),
    'r7_policy_preserves_whole_mask_loss': (
        math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('soft_dice_weight', float('nan'))),
            0.65, rel_tol=0.0, abs_tol=1e-12,
        )
        and math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('focal_bce_weight', float('nan'))),
            0.35, rel_tol=0.0, abs_tol=1e-12,
        )
    ),
    'r7_checkpoint_policy_is_decoupled': (
        r7_policy.get('checkpoint_policy', {}).get('save_checkpoint_on')
        == 'every strict numeric score improvement'
        and r7_policy.get('checkpoint_policy', {}).get('minimum_delta_use')
        == 'early-stopping patience reset only'
        and r7_policy.get('checkpoint_policy', {}).get(
            'minimum_delta_must_not_block_checkpoint_save'
        ) is True
    ),
    'r5_read_only_baseline_gate_matches': (
        r5_result_gate.get('pilot_version') == R5_PILOT_VERSION
        and r5_result_gate.get('code_revision') == R5_PILOT_CODE_REVISION
        and r5_result_gate.get('pilot_complete') is True
        and str(r5_result_gate.get('best_checkpoint_path'))
        == str(R5_BEST_CHECKPOINT_PATH)
        and r5_result_gate.get('best_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def r7_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


R7_INNER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_INNER_RADIUS_MM
)
R7_OUTER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_OUTER_RADIUS_MM
)


def r7_component_coefficient(epoch):
    epoch = int(epoch)
    if epoch < R7_COMPONENT_LOSS_START_EPOCH:
        return 0.0
    progress = (
        (epoch - R7_COMPONENT_LOSS_START_EPOCH + 1)
        / (R7_COMPONENT_LOSS_FULL_EPOCH - R7_COMPONENT_LOSS_START_EPOCH + 1)
    )
    return float(
        R7_COMPONENT_LOSS_MAX_COEFFICIENT * np.clip(progress, 0.0, 1.0)
    )


R7_COMPONENT_COEFFICIENT_SCHEDULE = {
    epoch: r7_component_coefficient(epoch)
    for epoch in range(1, MAX_EPOCHS + 1)
}
R7_EXPECTED_RAMP = {
    8: 0.02, 9: 0.04, 10: 0.06, 11: 0.08, 12: 0.10,
}
if not all(
    R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch] == 0.0
    for epoch in range(1, R7_COMPONENT_LOSS_START_EPOCH)
):
    raise RuntimeError('R7 local-contrastive coefficient is nonzero before epoch 8.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch], expected,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch, expected in R7_EXPECTED_RAMP.items()
):
    raise RuntimeError('R7 local-contrastive coefficient ramp changed.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch],
        R7_COMPONENT_LOSS_MAX_COEFFICIENT,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch in range(R7_COMPONENT_LOSS_FULL_EPOCH, MAX_EPOCHS + 1)
):
    raise RuntimeError('R7 local-contrastive coefficient does not remain capped.')


def r7_component_regions(exact_target, full_target, valid_domain):
    exact = np.asarray(exact_target, dtype=bool)
    full = np.asarray(full_target) > 0
    domain = np.asarray(valid_domain, dtype=bool)
    inner = ndimage.binary_dilation(exact, structure=R7_INNER_STRUCTURE)
    outer = ndimage.binary_dilation(exact, structure=R7_OUTER_STRUCTURE)
    negative = outer & ~inner & domain & ~full
    return inner, outer, negative


def pilot_extract_targets(
    row, local_origin, sample_type, atomic_component_id, component_tier
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary > 0
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(binary, dtype=np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if start_a != start_b or start_a != start_c:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_domain = F.max_pool3d(
        torch.from_numpy(domain_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0

    selected_sample = str(sample_type) == 'positive'
    selected_valid = (
        selected_sample and str(component_tier) in R7_SMALL_COMPONENT_TIERS
    )
    selected_component = np.zeros_like(component_support, dtype=bool)
    if selected_sample:
        labels, count = ndimage.label(
            component_support, structure=np.ones((3, 3, 3), dtype=bool)
        )
        expected_count = len(component_records.get(subject_id, []))
        if count != expected_count:
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {expected_count}.'
            )
        component_id = int(atomic_component_id)
        if component_id < 1 or component_id > count:
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is outside '
                f'the frozen 1..{count} range.'
            )
        selected_component = labels == component_id
        if not bool(selected_component.any()):
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is empty.'
            )

    selected_canvas, selected_start = symmetric_pad(
        selected_component.astype(np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if selected_start != start_a:
        raise RuntimeError(
            f'{subject_id}: selected-component padding start disagrees.'
        )
    selected_global = F.max_pool3d(
        torch.from_numpy(selected_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0
    selected_negative = np.zeros_like(selected_global, dtype=bool)
    if selected_valid:
        _, _, selected_negative = r7_component_regions(
            selected_global, global_target[0], global_domain
        )
        if not bool(selected_negative.any()):
            raise RuntimeError(
                f'{subject_id}: selected component has no lesion-free '
                '4-12 mm negative annulus.'
            )
    if selected_sample:
        frozen = [
            item for item in component_records[subject_id]
            if int(item['atomic_component_id']) == int(atomic_component_id)
        ]
        if len(frozen) != 1:
            raise RuntimeError(
                f'{subject_id}: selected component does not map uniquely to '
                'the frozen component audit.'
            )
        expected_global_voxels = int(frozen[0]['global_proposal_voxels'])
        if int(selected_global.sum()) != expected_global_voxels:
            raise RuntimeError(
                f'{subject_id}: selected component has '
                f'{int(selected_global.sum())} pooled voxels; expected '
                f'{expected_global_voxels}.'
            )
        if bool((selected_negative & selected_global).any()):
            raise RuntimeError(
                f'{subject_id}: negative annulus overlaps exact component.'
            )
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'selected_component_target': selected_global[None].astype(bool),
        'selected_component_negative_annulus': (
            selected_negative[None].astype(bool)
        ),
        'selected_component_valid': np.asarray(
            selected_valid, dtype=bool
        ),
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'selected_component_target': targets[
                'selected_component_target'
            ],
            'selected_component_negative_annulus': targets[
                'selected_component_negative_annulus'
            ],
            'selected_component_valid': targets[
                'selected_component_valid'
            ],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def r7_local_contrastive_loss(
    logits, exact_component_target, negative_annulus_target, component_valid,
):
    exact_target = exact_component_target.bool()
    negative_target = negative_annulus_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    positive_losses = []
    negative_losses = []
    rank_losses = []
    positive_logits = []
    hard_negative_logits = []
    selected_negative_counts = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]].float()
        negative = logits[batch_index][negative_target[batch_index]].float()
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError(
                'A valid r7 component must have positive voxels and a '
                'lesion-free negative annulus.'
            )
        hard_count = min(
            int(negative.numel()),
            max(
                R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * int(positive.numel()),
            ),
        )
        hard_negative = torch.topk(
            negative, k=hard_count, largest=True
        ).values
        positive_loss = F.softplus(-positive).mean()
        negative_loss = F.softplus(hard_negative).mean()
        rank_loss = F.softplus(
            R7_LOGIT_MARGIN - positive.mean() + hard_negative.mean()
        )
        contrastive_loss = (
            R7_POSITIVE_WEIGHT * positive_loss
            + R7_HARD_NEGATIVE_WEIGHT * negative_loss
            + R7_RANK_WEIGHT * rank_loss
        )
        losses.append(contrastive_loss)
        positive_losses.append(positive_loss)
        negative_losses.append(negative_loss)
        rank_losses.append(rank_loss)
        positive_logits.append(positive.mean())
        hard_negative_logits.append(hard_negative.mean())
        selected_negative_counts.append(hard_count)

    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'local_contrastive_loss': zero,
            'component_positive_loss': zero,
            'component_hard_negative_loss': zero,
            'component_rank_loss': zero,
            'mean_component_positive_logit': zero.detach(),
            'mean_component_hard_negative_logit': zero.detach(),
            'selected_hard_negative_voxels': zero.detach(),
            'selected_components_in_batch': zero.detach(),
        }
    return torch.stack(losses).mean(), {
        'local_contrastive_loss': torch.stack(losses).mean(),
        'component_positive_loss': torch.stack(positive_losses).mean(),
        'component_hard_negative_loss': torch.stack(negative_losses).mean(),
        'component_rank_loss': torch.stack(rank_losses).mean(),
        'mean_component_positive_logit': torch.stack(positive_logits).mean(),
        'mean_component_hard_negative_logit': torch.stack(
            hard_negative_logits
        ).mean(),
        'selected_hard_negative_voxels': torch.as_tensor(
            float(sum(selected_negative_counts)), device=logits.device
        ),
        'selected_components_in_batch': torch.as_tensor(
            float(len(losses)), device=logits.device
        ),
    }


def r7_local_contrastive_proposal_loss(
    logits, full_target, exact_component_target,
    negative_annulus_target, component_valid, epoch,
):
    whole_dice = pilot_soft_dice_loss(logits, full_target).mean()
    whole_bce = pilot_focal_bce_loss(logits, full_target).mean()
    whole_loss = 0.65 * whole_dice + 0.35 * whole_bce
    contrastive_loss, contrastive_parts = r7_local_contrastive_loss(
        logits, exact_component_target, negative_annulus_target,
        component_valid,
    )
    coefficient = r7_component_coefficient(epoch)
    proposal_loss = whole_loss + coefficient * contrastive_loss
    return proposal_loss, {
        'proposal_whole_mask_loss': whole_loss,
        'proposal_dice_loss': whole_dice,
        'proposal_focal_bce_loss': whole_bce,
        **contrastive_parts,
        'component_loss_coefficient': torch.as_tensor(
            coefficient, device=logits.device, dtype=torch.float32
        ),
    }


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(
    model, cpu_batch, device, epoch, batch_index,
    dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    if dropout_flags_override is not None:
        required_flags = {
            'force_atlas_dropout', 'force_vascular_dropout',
            'force_occlusion_dropout', 'force_clinical_dropout',
            'force_global_dropout', 'force_arterial_dropout',
        }
        if set(dropout_flags_override) != required_flags:
            raise RuntimeError(
                'Dropout override keys do not match the frozen branch set.'
            )
        flags = {
            name: bool(dropout_flags_override[name])
            for name in sorted(required_flags)
        }
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_loss, proposal_parts = r7_local_contrastive_proposal_loss(
        output['global_proposal_logits'], global_target,
        batch['selected_component_target'],
        batch['selected_component_negative_annulus'],
        batch['selected_component_valid'], epoch,
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        **proposal_parts,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'arterial_valid_nodes': arterial_valid.sum().detach().float(),
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'arterial_dropped': torch.as_tensor(float(flags['force_arterial_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_volume_bounded_prediction(probability, target_voxels):
    probability = np.asarray(probability, dtype=np.float32)
    budget = min(
        probability.size,
        max(1, int(math.ceil(
            R7_VOLUME_BUDGET_MULTIPLIER * int(target_voxels)
        ))),
    )
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape), budget


def pilot_histogram_average_precision(positive_histogram, negative_histogram):
    positive = np.asarray(positive_histogram, dtype=np.float64)[::-1]
    negative = np.asarray(negative_histogram, dtype=np.float64)[::-1]
    total_positive = float(positive.sum())
    if total_positive <= 0:
        return float('nan')
    cumulative_positive = np.cumsum(positive)
    cumulative_total = np.cumsum(positive + negative)
    precision = cumulative_positive / np.maximum(cumulative_total, 1.0)
    return float((precision * positive).sum() / total_positive)


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    voxel_precision = total_intersection / max(total_prediction, 1)
    predicted_to_target_volume_ratio = (
        total_prediction / max(total_target, 1)
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'voxel_precision': float(voxel_precision),
        'predicted_to_target_volume_ratio': float(
            predicted_to_target_volume_ratio
        ),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['volume_calibration_score'] = np.exp(-np.abs(
        np.log(np.maximum(
            proposal_threshold_df['predicted_to_target_volume_ratio'], 1e-6
        ))
    ))
    proposal_threshold_df['proposal_selection_score'] = (
        0.35 * proposal_threshold_df['positive_global_dice']
        + 0.15 * proposal_threshold_df['lesion_f1']
        + 0.20 * proposal_threshold_df['voxel_precision']
        + 0.15 * proposal_threshold_df['component_recall']
        + 0.15 * proposal_threshold_df['volume_calibration_score']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        [
            'proposal_selection_score', 'positive_global_dice',
            'voxel_precision', 'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    proposal_ap_positive_histogram = np.zeros(256, dtype=np.int64)
    proposal_ap_negative_histogram = np.zeros(256, dtype=np.int64)
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        proposal_intersection = int((proposal_prediction & target).sum())
        proposal_predicted_voxels = int(proposal_prediction.sum())
        proposal_distance = None
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        budgeted_proposal_prediction = None
        budgeted_proposal_distance = None
        proposal_volume_budget_voxels = 0
        if hard_eligible and target_voxels > 0:
            (
                budgeted_proposal_prediction,
                proposal_volume_budget_voxels,
            ) = pilot_volume_bounded_prediction(proposal, target_voxels)
            budgeted_proposal_distance = ndimage.distance_transform_edt(
                ~budgeted_proposal_prediction, sampling=case['spacing']
            )
            proposal_ap_positive_histogram += np.histogram(
                proposal[target], bins=256, range=(0.0, 1.0)
            )[0]
            proposal_ap_negative_histogram += np.histogram(
                proposal[~target], bins=256, range=(0.0, 1.0)
            )[0]
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
            'proposal_predicted_voxels': proposal_predicted_voxels,
            'proposal_intersection_voxels': proposal_intersection,
            'proposal_voxel_precision': (
                proposal_intersection / max(proposal_predicted_voxels, 1)
            ),
            'proposal_voxel_recall': (
                proposal_intersection / max(target_voxels, 1)
            ),
            'proposal_predicted_to_target_volume_ratio': (
                proposal_predicted_voxels / max(target_voxels, 1)
            ),
            'proposal_volume_budget_voxels': proposal_volume_budget_voxels,
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            budgeted_proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    budgeted_proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                budgeted_proposal_max_iou = float(
                    budgeted_proposal_iou[component_id][
                        'maximum_component_iou'
                    ]
                )
                proposal_distance_mm = float('nan')
                if proposal_distance is not None:
                    proposal_distance_mm = float(
                        proposal_distance[component_slice][component_local].min(
                            initial=np.inf
                        )
                    )
                budgeted_proposal_distance_mm = float(
                    budgeted_proposal_distance[component_slice][
                        component_local
                    ].min(initial=np.inf)
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                budgeted_proposal_covered = (
                    budgeted_proposal_max_iou >= 0.05
                )
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += int(budgeted_proposal_covered)
                tier_counts[tier][3] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'proposal_minimum_distance_mm': proposal_distance_mm,
                    'budgeted_proposal_maximum_component_iou': (
                        budgeted_proposal_max_iou
                    ),
                    'budgeted_proposal_minimum_distance_mm': (
                        budgeted_proposal_distance_mm
                    ),
                    'proposal_within_2mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 2.0
                    ),
                    'proposal_within_4mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 4.0
                    ),
                    'proposal_within_8mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_within_2mm': bool(
                        budgeted_proposal_distance_mm <= 2.0
                    ),
                    'budgeted_proposal_within_4mm': bool(
                        budgeted_proposal_distance_mm <= 4.0
                    ),
                    'budgeted_proposal_within_8mm': bool(
                        budgeted_proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_covered': bool(
                        budgeted_proposal_covered
                    ),
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, budgeted_proposal_tp, total = tier_counts[tier]
        tier_components = [
            item for item in component_rows
            if item['component_tier'] == tier
        ]
        finite_distances = [
            float(item['proposal_minimum_distance_mm'])
            for item in tier_components
            if np.isfinite(item['proposal_minimum_distance_mm'])
        ]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
            'budgeted_proposal_component_recall': (
                budgeted_proposal_tp / max(total, 1)
            ),
            'proposal_within_2mm_fraction': float(np.mean([
                item['proposal_within_2mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_4mm_fraction': float(np.mean([
                item['proposal_within_4mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_8mm_fraction': float(np.mean([
                item['proposal_within_8mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_2mm_fraction': float(np.mean([
                item['budgeted_proposal_within_2mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_4mm_fraction': float(np.mean([
                item['budgeted_proposal_within_4mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_8mm_fraction': float(np.mean([
                item['budgeted_proposal_within_8mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'median_finite_proposal_distance_mm': float(
                np.median(finite_distances)
            ) if finite_distances else float('nan'),
            'mean_proposal_maximum_probability': float(np.mean([
                item['proposal_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'mean_final_maximum_probability': float(np.mean([
                item['final_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'representation_failure_fraction': float(np.mean([
                item['representation_failure_below_0p10']
                for item in tier_components
            ])) if tier_components else 0.0,
        })
    small_component_rows = [
        item for item in component_rows
        if item['component_tier'] in set(COMPONENT_TIERS[:4])
    ]
    proposal_voxel_average_precision = pilot_histogram_average_precision(
        proposal_ap_positive_histogram, proposal_ap_negative_histogram
    )
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'proposal_voxel_precision': float(
            proposal_selected['voxel_precision']
        ),
        'proposal_predicted_to_target_volume_ratio': float(
            proposal_selected['predicted_to_target_volume_ratio']
        ),
        'proposal_volume_calibration_score': float(
            proposal_selected['volume_calibration_score']
        ),
        'proposal_voxel_average_precision_256bin': float(
            proposal_voxel_average_precision
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'small_components_audited': len(small_component_rows),
        'small_component_proposal_within_2mm_fraction': float(np.mean([
            item['proposal_within_2mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_4mm_fraction': float(np.mean([
            item['proposal_within_4mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_8mm_fraction': float(np.mean([
            item['proposal_within_8mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_budgeted_proposal_within_2mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_2mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_4mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_4mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_8mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_8mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered'] for item in component_rows
        ])) if component_rows else 0.0,
        'small_budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_representation_failure_fraction': float(np.mean([
            item['representation_failure_below_0p10']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    required_finite_summary_metrics = [
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'proposal_global_dice', 'proposal_component_recall',
        'proposal_voxel_precision',
        'proposal_predicted_to_target_volume_ratio',
        'proposal_voxel_average_precision_256bin',
        'budgeted_proposal_component_recall',
        'small_budgeted_proposal_component_recall',
        'small_component_budgeted_proposal_within_2mm_fraction',
        'small_component_budgeted_proposal_within_4mm_fraction',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'small_component_representation_failure_fraction',
        'pilot_checkpoint_score',
    ]
    nonfinite_summary_metrics = [
        name for name in required_finite_summary_metrics
        if not np.isfinite(float(summary[name]))
    ]
    if nonfinite_summary_metrics:
        raise RuntimeError(
            'Validation produced non-finite required metrics: '
            + ', '.join(nonfinite_summary_metrics)
        )
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
            and completed.get('best_checkpoint_sha256')
            == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R7 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_index = None
    smoke_subject_id = None
    for candidate_index, entry in enumerate(dataset.schedule):
        if entry['sample_type'] != 'positive':
            continue
        candidate_subject = str(entry['subject_id'])
        if not as_bool(
            fit_row_by_subject[candidate_subject]['trial8_atlas_enabled']
        ):
            continue
        candidate_sample = dataset[candidate_index]
        candidate_checks = {
            'atlas_enabled': bool(np.asarray(
                candidate_sample['atlas_enabled']
            ).reshape(-1)[0]),
            'territory_nodes_valid': bool(np.asarray(
                candidate_sample['territory_valid']
            ).astype(bool).any()),
            'vessel_nodes_valid': bool(np.asarray(
                candidate_sample['vessel_valid']
            ).astype(bool).any()),
            'selected_component_valid': bool(np.asarray(
                candidate_sample['selected_component_valid']
            ).reshape(-1)[0]),
        }
        if all(candidate_checks.values()):
            smoke_index = int(candidate_index)
            smoke_subject_id = candidate_subject
            break
    if smoke_index is None:
        raise RuntimeError(
            'No positive fold-0-fit sample has simultaneous atlas, territory, '
            'vessel, and selected-component support for the all-branch smoke '
            'test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    smoke_dropout_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device,
            epoch=R7_COMPONENT_LOSS_FULL_EPOCH, batch_index=0,
            dropout_flags_override=smoke_dropout_flags,
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    r7_smoke_checks = {
        'positive_sample_selected': bool(
            smoke_batch['selected_component_valid'].bool().all().item()
        ),
        'atlas_enabled_for_smoke': bool(
            smoke_batch['atlas_enabled'].bool().all().item()
        ),
        'territory_support_for_smoke': bool(
            smoke_batch['territory_valid'].bool().any().item()
        ),
        'vessel_support_for_smoke': bool(
            smoke_batch['vessel_valid'].bool().any().item()
        ),
        'projected_arterial_nodes_valid_for_smoke': float(
            smoke_parts['arterial_valid_nodes'].detach().float().cpu()
        ) > 0.0,
        'arterial_loss_finite_for_smoke': bool(torch.isfinite(
            smoke_parts['arterial_loss']
        ).item()),
        'all_branch_dropouts_disabled_for_smoke': not any(
            smoke_dropout_flags.values()
        ),
        'exact_component_target_nonempty': bool(
            smoke_batch['selected_component_target'].bool().any().item()
        ),
        'negative_annulus_nonempty': bool(
            smoke_batch['selected_component_negative_annulus']
            .bool().any().item()
        ),
        'negative_annulus_disjoint_from_exact_component': not bool(torch.any(
            smoke_batch['selected_component_target'].bool()
            & smoke_batch['selected_component_negative_annulus'].bool()
        ).item()),
        'local_contrastive_loss_positive': float(
            smoke_parts['local_contrastive_loss']
            .detach().float().cpu()
        ) > 0.0,
        'smoke_uses_maximum_contrastive_coefficient': math.isclose(
            float(smoke_parts['component_loss_coefficient']
                  .detach().float().cpu()),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT,
            rel_tol=0.0, abs_tol=1e-7,
        ),
        'whole_mask_proposal_loss_finite': bool(torch.isfinite(
            smoke_parts['proposal_whole_mask_loss']
        ).item()),
    }
    if not all(r7_smoke_checks.values()):
        failed = [
            name for name, passed in r7_smoke_checks.items() if not passed
        ]
        raise RuntimeError(
            'R7 local-contrastive smoke checks failed: ' + ', '.join(failed)
        )
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    if R7_R5_BASELINE_DIAGNOSTIC_PATH.exists():
        r5_baseline_diagnostic = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
        if (
            r5_baseline_diagnostic.get('r5_checkpoint_sha256')
            != r5_checkpoint_sha256
            or r5_baseline_diagnostic.get('metric_contract_revision')
            != PILOT_CODE_REVISION
        ):
            raise RuntimeError(
                'Existing r5 volume-bounded baseline is incompatible with '
                'the current r7 contract.'
            )
        r5_recomputed_summary = dict(
            r5_baseline_diagnostic['validation_summary']
        )
    else:
        rng_state_before_baseline = pilot_capture_rng_state()
        r5_checkpoint = torch.load(
            R5_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
        )
        r5_checkpoint_checks = {
            'pilot_version': (
                r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
            ),
            'code_revision': (
                r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
            ),
            'parameter_count': int(
                r5_checkpoint.get('parameter_count', -1)
            ) == EXPECTED_V22_PARAMETERS,
        }
        if not all(r5_checkpoint_checks.values()):
            failed = [
                name for name, passed in r5_checkpoint_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Unsafe r5 read-only baseline checkpoint: '
                + ', '.join(failed)
            )
        r5_baseline_model = VascuFateNetV22(pilot_schema).to(device).eval()
        r5_baseline_model.load_state_dict(r5_checkpoint['ema_model_state'])
        for parameter in r5_baseline_model.parameters():
            parameter.requires_grad_(False)
        print(
            '\nRecomputing the completed r5 best checkpoint with the r7 '
            'precision- and volume-bounded diagnostic contract.'
        )
        r5_recomputed_summary, r5_recomputed_tiers = pilot_validate(
            r5_baseline_model, epoch=0, device=device
        )
        r5_baseline_diagnostic = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'metric_contract_revision': PILOT_CODE_REVISION,
            'r5_pilot_version': R5_PILOT_VERSION,
            'r5_code_revision': R5_PILOT_CODE_REVISION,
            'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'validation_summary': r5_recomputed_summary,
            'component_tier_metrics': r5_recomputed_tiers,
            'r5_checkpoint_loaded_into_r7_model': False,
            'optimizer_steps': 0,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        }
        atomic_write_json(
            r5_baseline_diagnostic, R7_R5_BASELINE_DIAGNOSTIC_PATH
        )
        del r5_baseline_model, r5_checkpoint
        gc.collect()
        torch.cuda.empty_cache()
        pilot_restore_rng_state(rng_state_before_baseline)

    
    
    
    
    del model, ema_model
    gc.collect()
    torch.cuda.empty_cache()
    set_determinism(PILOT_SEED)
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed during lineage reset.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    lineage_smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True
    )
    lineage_smoke_batch = next(iter(lineage_smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        lineage_smoke_loss, lineage_smoke_parts = (
            pilot_compute_training_loss(
                model, lineage_smoke_batch, device, epoch=1, batch_index=0
            )
        )
    if not bool(torch.isfinite(lineage_smoke_loss).item()):
        raise RuntimeError('R7 controlled-lineage smoke loss is non-finite.')
    if float(
        lineage_smoke_parts['component_loss_coefficient']
        .detach().float().cpu()
    ) != 0.0:
        raise RuntimeError(
            'R7 controlled-lineage epoch-1 coefficient is not exactly zero.'
        )
    lineage_smoke_loss.backward()
    lineage_gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            parameter.grad
            for name, parameter in model.named_parameters()
            if name.startswith(prefix) and parameter.grad is not None
        ]
        lineage_gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(gradient).all().item())
            for gradient in gradients
        ) and any(
            float(gradient.detach().float().norm().cpu()) > 0.0
            for gradient in gradients
        )
    model.zero_grad(set_to_none=True)
    if not all(lineage_gradient_checks.values()):
        failed = [
            name for name, passed in lineage_gradient_checks.items()
            if not passed
        ]
        raise RuntimeError(
            'R7 controlled-lineage gradient smoke failed: '
            + ', '.join(failed)
        )

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r7_last_checkpoint_only',
        'strict_best_checkpoint_policy': (
            'save every strict numeric pilot-score improvement'
        ),
        'minimum_delta_policy': (
            'used only to reset early-stopping patience'
        ),
        'r7_preflight_gate_sha256': file_sha256(R7_PREFLIGHT_GATE_PATH),
        'r7_proposal_loss_policy_sha256': file_sha256(R7_POLICY_PATH),
        'component_loss_start_epoch': R7_COMPONENT_LOSS_START_EPOCH,
        'component_loss_full_epoch': R7_COMPONENT_LOSS_FULL_EPOCH,
        'component_loss_maximum_coefficient': (
            R7_COMPONENT_LOSS_MAX_COEFFICIENT
        ),
        'component_positive_weight': R7_POSITIVE_WEIGHT,
        'component_hard_negative_weight': R7_HARD_NEGATIVE_WEIGHT,
        'component_rank_weight': R7_RANK_WEIGHT,
        'component_inner_radius_mm': R7_INNER_RADIUS_MM,
        'component_outer_radius_mm': R7_OUTER_RADIUS_MM,
        'component_logit_margin': R7_LOGIT_MARGIN,
        'component_eligible_tiers': sorted(R7_SMALL_COMPONENT_TIERS),
        'component_coefficient_schedule': {
            str(epoch): coefficient
            for epoch, coefficient in R7_COMPONENT_COEFFICIENT_SCHEDULE.items()
        },
        'proposal_volume_budget_multiplier': R7_VOLUME_BUDGET_MULTIPLIER,
        'proposal_volume_budget_policy': (
            'diagnostic only: retain the top 3x ground-truth target voxels; '
            'never used as an inference rule or checkpoint-selection input'
        ),
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'controlled_lineage_matches_r5_seed': True,
        'controlled_lineage_epoch1_component_coefficient': 0.0,
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'controlled_lineage_gradient_checks': lineage_gradient_checks,
        'controlled_lineage_smoke_loss': float(
            lineage_smoke_loss.detach().cpu()
        ),
        'r7_selected_component_smoke_checks': r7_smoke_checks,
        'r7_component_coefficient_schedule_verified': True,
        'r7_smoke_subject_id': smoke_subject_id,
        'r7_smoke_schedule_index': smoke_index,
        'r7_smoke_dropout_flags': smoke_dropout_flags,
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R7 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R7 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r7 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Controlled r5-seed lineage:          verified')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('Local contrastive rescue:              small tiers, epochs 8-40')
    print('Maximum contrastive coefficient:       0.10')
    print('Proposal anti-shortcut diagnostics:    oracle-volume-bounded (audit only)')
    print('R5 recomputed mean/global Dice:        '
          f'{r5_recomputed_summary["positive_mean_dice"]:.4f} / '
          f'{r5_recomputed_summary["positive_global_dice"]:.4f}')
    print('R5 oracle-budgeted small within 8 mm:  '
          f'{r5_recomputed_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}')
    print('Best-checkpoint minimum delta:        disabled (strict numeric best)')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        strict_best_improved = current_score > best_score
        patience_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if strict_best_improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
        if patience_improved:
            patience_reference_score = current_score
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        epoch_tier_metrics = {
            str(row['component_tier']): row for row in tier_rows
        }
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'proposal_voxel_precision': summary['proposal_voxel_precision'],
            'proposal_predicted_to_target_volume_ratio': summary[
                'proposal_predicted_to_target_volume_ratio'
            ],
            'proposal_voxel_average_precision_256bin': summary[
                'proposal_voxel_average_precision_256bin'
            ],
            'budgeted_proposal_component_recall': summary[
                'budgeted_proposal_component_recall'
            ],
            'small_component_proposal_within_2mm_fraction': summary[
                'small_component_proposal_within_2mm_fraction'
            ],
            'small_component_proposal_within_4mm_fraction': summary[
                'small_component_proposal_within_4mm_fraction'
            ],
            'small_component_proposal_within_8mm_fraction': summary[
                'small_component_proposal_within_8mm_fraction'
            ],
            'small_component_budgeted_proposal_within_8mm_fraction': summary[
                'small_component_budgeted_proposal_within_8mm_fraction'
            ],
            'small_component_representation_failure_fraction': summary[
                'small_component_representation_failure_fraction'
            ],
            **{
                f'{tier_name}_{metric_name}': float(
                    epoch_tier_metrics.get(tier_name, {}).get(
                        metric_name, float('nan')
                    )
                )
                for tier_name in COMPONENT_TIERS
                for metric_name in (
                    'final_component_recall',
                    'proposal_component_recall',
                    'budgeted_proposal_component_recall',
                    'proposal_within_8mm_fraction',
                    'budgeted_proposal_within_8mm_fraction',
                    'representation_failure_fraction',
                )
            },
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'strict_best_improved': strict_best_improved,
            'patience_improved_by_minimum_delta': patience_improved,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if strict_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'contrastive={running["local_contrastive_loss"]/max(processed,1):.4f}, '
            f'coef={running["component_loss_coefficient"]/max(processed,1):.3f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        print(
            '  R7 raw proposal geometry: small within 2/4/8 mm='
            f'{summary["small_component_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_8mm_fraction"]:.4f}, '
            'small representation failure='
            f'{summary["small_component_representation_failure_fraction"]:.4f}'
        )
        print(
            '  Oracle-volume-bounded diagnostic: small within 2/4/8 mm='
            f'{summary["small_component_budgeted_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}, '
            f'component recall={summary["budgeted_proposal_component_recall"]:.4f}, '
            f'precision/AP={summary["proposal_voxel_precision"]:.4f}/'
            f'{summary["proposal_voxel_average_precision_256bin"]:.4f}, '
            f'volume ratio={summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_baseline = r7_policy['r5_baseline']
    success_policy = r7_policy['r7_directional_success_gates']
    best_tier_by_name = {
        str(row['component_tier']): row for row in best_tier_rows
    }
    best_large_proposal_recall = float(
        best_tier_by_name.get('large', {}).get(
            'proposal_component_recall', 0.0
        )
    )
    mean_dice_improvement = (
        float(best_summary['positive_mean_dice'])
        - float(r5_recomputed_summary['positive_mean_dice'])
    )
    budgeted_small_within_8mm_improvement = (
        float(best_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
        - float(r5_recomputed_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
    )
    small_representation_failure_reduction = (
        float(r5_recomputed_summary[
            'small_component_representation_failure_fraction'
        ])
        - float(best_summary[
            'small_component_representation_failure_fraction'
        ])
    )
    global_dice_regression = (
        float(r5_recomputed_summary['positive_global_dice'])
        - float(best_summary['positive_global_dice'])
    )
    proposal_dice_regression = (
        float(r5_recomputed_summary['proposal_global_dice'])
        - float(best_summary['proposal_global_dice'])
    )
    r7_success_checks = {
        'mean_dice_does_not_materially_regress': (
            -mean_dice_improvement
            <= float(success_policy[
                'mean_dice_maximum_allowed_regression'
            ])
        ),
        'mean_dice_reaches_target_improvement': (
            mean_dice_improvement
            >= float(success_policy[
                'mean_dice_target_absolute_improvement'
            ])
        ),
        'volume_bounded_small_within_8mm_improves': (
            budgeted_small_within_8mm_improvement
            >= float(success_policy[
                'volume_bounded_small_within_8mm_absolute_improvement_at_least'
            ])
        ),
        'small_representation_failure_reduced_by_required_margin': (
            small_representation_failure_reduction
            >= float(success_policy[
                'small_representation_failure_absolute_reduction_at_least'
            ])
        ),
        'proposal_component_recall_meets_minimum': (
            float(best_summary['proposal_component_recall'])
            >= float(success_policy['proposal_component_recall_minimum'])
        ),
        'global_dice_regression_within_limit': (
            global_dice_regression
            <= float(success_policy[
                'global_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_dice_regression_within_limit': (
            proposal_dice_regression
            <= float(success_policy[
                'proposal_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_volume_ratio_is_bounded': (
            float(best_summary[
                'proposal_predicted_to_target_volume_ratio'
            ])
            <= float(success_policy[
                'proposal_predicted_to_target_volume_ratio_maximum'
            ])
        ),
        'empty_false_positive_volume_within_limit': (
            float(best_summary['empty_false_positive_volume_ml'])
            <= float(success_policy[
                'mean_false_positive_volume_ml_maximum'
            ])
        ),
        'large_component_proposal_recall_meets_minimum': (
            best_large_proposal_recall
            >= float(success_policy[
                'large_component_proposal_recall_minimum'
            ])
        ),
    }
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'r5_baseline_evidence': r5_baseline,
        'r5_recomputed_under_r7_metric_contract': r5_recomputed_summary,
        'r5_read_only_baseline_diagnostic_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r7_directional_success_policy': success_policy,
        'r7_directional_success_checks': r7_success_checks,
        'r7_directional_success_passed': all(r7_success_checks.values()),
        'r7_directional_effects': {
            'mean_dice_absolute_change': mean_dice_improvement,
            'volume_bounded_small_within_8mm_absolute_change': (
                budgeted_small_within_8mm_improvement
            ),
            'small_representation_failure_absolute_reduction': (
                small_representation_failure_reduction
            ),
            'global_dice_regression': global_dice_regression,
            'proposal_dice_regression': proposal_dice_regression,
            'proposal_predicted_to_target_volume_ratio': float(
                best_summary['proposal_predicted_to_target_volume_ratio']
            ),
            'proposal_voxel_average_precision_256bin': float(
                best_summary['proposal_voxel_average_precision_256bin']
            ),
            'large_component_proposal_recall': (
                best_large_proposal_recall
            ),
        },
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(
        'Raw small proposal within 8 mm:   '
        f'{best_summary["small_component_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Oracle-budgeted small within 8 mm: '
        f'{best_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Proposal precision / AP:          '
        f'{best_summary["proposal_voxel_precision"]:.4f} / '
        f'{best_summary["proposal_voxel_average_precision_256bin"]:.4f}'
    )
    print(
        'Proposal predicted/target ratio:  '
        f'{best_summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
    )
    print(
        'Mean Dice change vs recomputed r5:'
        f' {mean_dice_improvement:+.4f}'
    )
    print(
        'Budgeted within-8-mm change vs r5:'
        f' {budgeted_small_within_8mm_improvement:+.4f}'
    )
    print(
        'Small representation failure:     '
        f'{best_summary["small_component_representation_failure_fraction"]:.4f}'
    )
    print(
        'R7 directional success gates:     '
        f'{all(r7_success_checks.values())}'
    )
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')






R7D_EXPOSURE_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure.csv'
R7D_EXPOSURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure_by_tier.csv'
)
R7D_RESOLUTION_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_component_audit.csv'
)
R7D_RESOLUTION_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_summary.csv'
)
R7D_PAIRED_CASE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_case_metrics.csv'
)
R7D_PAIRED_COMPONENT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_component_metrics.csv'
)
R7D_MARGIN_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_train_heldout_margin_audit.csv'
)
R7D_MARGIN_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_margin_summary.csv'
)
R7D_VOLUME_BUDGET_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_audit.csv'
)
R7D_VOLUME_BUDGET_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_summary.csv'
)
R7D_CHECKPOINT_PARETO_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'checkpoint_metric_pareto.csv'
)
R7D_BRANCH_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'branch_reliance_comparison.csv'
)
R7D_VISUAL_CANDIDATES_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'visual_qa_candidates.csv'
)
R7D_CHECKS_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'diagnostic_checks.csv'
R7D_TARGET_ACCESS_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'development_target_access_audit.csv'
)

R7D_OUTPUT_PATHS = [
    R7D_EXPOSURE_PATH,
    R7D_EXPOSURE_SUMMARY_PATH,
    R7D_RESOLUTION_PATH,
    R7D_RESOLUTION_SUMMARY_PATH,
    R7D_PAIRED_CASE_PATH,
    R7D_PAIRED_COMPONENT_PATH,
    R7D_MARGIN_PATH,
    R7D_MARGIN_SUMMARY_PATH,
    R7D_VOLUME_BUDGET_PATH,
    R7D_VOLUME_BUDGET_SUMMARY_PATH,
    R7D_CHECKPOINT_PARETO_PATH,
    R7D_BRANCH_PATH,
    R7D_VISUAL_CANDIDATES_PATH,
    R7D_CHECKS_PATH,
    R7D_TARGET_ACCESS_PATH,
    R7_FAILURE_SUMMARY_PATH,
]


def r7d_existing_result_is_complete():
    if not R7_FAILURE_GATE_PATH.exists():
        return False
    gate = load_json(R7_FAILURE_GATE_PATH)
    if not (
        gate.get('diagnostic_complete') is True
        and gate.get('diagnostic_version') == R7_FAILURE_DIAGNOSTIC_VERSION
        and gate.get('code_revision') == R7_FAILURE_DIAGNOSTIC_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_checkpoint_sha256')
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    ):
        return False
    output_hashes = gate.get('output_sha256', {})
    return all(
        path.exists()
        and output_hashes.get(path.name) == file_sha256(path)
        for path in R7D_OUTPUT_PATHS
    )


def r7d_gini(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0 or float(values.sum()) <= 0.0:
        return 0.0
    values = np.sort(values)
    index = np.arange(1, values.size + 1, dtype=np.float64)
    return float(
        np.sum((2.0 * index - values.size - 1.0) * values)
        / (values.size * values.sum())
    )


def r7d_component_support(row):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    mode = str(row['trial8_supervision_mode'])
    if mode == 'hard_and_soft':
        support = binary
    elif mode == 'soft_only_subvoxel':
        support = soft > 0
    elif mode == 'true_empty':
        support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unsupported supervision mode {mode!r}.'
        )
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: non-finite target arrays.')
    return binary, soft, support, spacing


def r7d_component_schedule_exposure():
    counts = defaultdict(int)
    first_epoch = {}
    last_epoch = {}
    for epoch in range(R7_COMPONENT_LOSS_START_EPOCH, MAX_EPOCHS + 1):
        schedule = pilot_build_epoch_schedule(epoch)
        for entry in schedule:
            if (
                str(entry['sample_type']) != 'positive'
                or str(entry['component_tier']) not in R7_SMALL_COMPONENT_TIERS
            ):
                continue
            key = (
                str(entry['subject_id']), int(entry['atomic_component_id'])
            )
            counts[key] += 1
            first_epoch.setdefault(key, int(epoch))
            last_epoch[key] = int(epoch)

    rows = []
    for item in component_df.sort_values(
        ['subject_id', 'atomic_component_id']
    ).to_dict('records'):
        key = (str(item['subject_id']), int(item['atomic_component_id']))
        tier = str(item['component_tier'])
        eligible = tier in R7_SMALL_COMPONENT_TIERS
        selections = int(counts.get(key, 0)) if eligible else 0
        rows.append({
            'subject_id': key[0],
            'atomic_component_id': key[1],
            'component_tier': tier,
            'component_voxels': int(item['component_voxels']),
            'component_volume_ml': float(item['component_volume_ml']),
            'global_proposal_voxels': int(item['global_proposal_voxels']),
            'local_contrastive_eligible': bool(eligible),
            'epochs_eligible': (
                MAX_EPOCHS - R7_COMPONENT_LOSS_START_EPOCH + 1
                if eligible else 0
            ),
            'selections_epochs_8_to_40': selections,
            'first_selected_epoch': first_epoch.get(key, np.nan),
            'last_selected_epoch': last_epoch.get(key, np.nan),
            'never_selected': bool(eligible and selections == 0),
            'selected_once': bool(eligible and selections == 1),
            'selected_multiple_times': bool(eligible and selections > 1),
        })
    frame = pd.DataFrame(rows)
    summary_rows = []
    for tier in COMPONENT_TIERS:
        group = frame[frame['component_tier'].astype(str) == tier]
        eligible = group[group['local_contrastive_eligible'].astype(bool)]
        values = eligible['selections_epochs_8_to_40'].to_numpy(
            dtype=np.float64
        )
        summary_rows.append({
            'component_tier': tier,
            'components': int(len(group)),
            'eligible_components': int(len(eligible)),
            'never_selected_components': int(
                eligible['never_selected'].sum()
            ),
            'never_selected_fraction': float(
                eligible['never_selected'].mean()
            ) if len(eligible) else 0.0,
            'selected_once_fraction': float(
                eligible['selected_once'].mean()
            ) if len(eligible) else 0.0,
            'minimum_selections': int(values.min()) if len(values) else 0,
            'median_selections': float(np.median(values)) if len(values) else 0.0,
            'mean_selections': float(values.mean()) if len(values) else 0.0,
            'maximum_selections': int(values.max()) if len(values) else 0,
            'selection_gini': r7d_gini(values),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(frame, R7D_EXPOSURE_PATH)
    atomic_write_csv(summary, R7D_EXPOSURE_SUMMARY_PATH)
    return frame, summary


def r7d_maximum_component_cell_matching(cell_sets):
    matched_cell = {}

    def visit(component_index, seen):
        for cell_id in sorted(cell_sets[component_index]):
            if cell_id in seen:
                continue
            seen.add(cell_id)
            if (
                cell_id not in matched_cell
                or visit(matched_cell[cell_id], seen)
            ):
                matched_cell[cell_id] = component_index
                return True
        return False

    matched = 0
    for component_index in range(len(cell_sets)):
        matched += int(visit(component_index, set()))
    return int(matched)


def r7d_resolution_audit():
    component_rows = []
    subject_rows = []
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    canvas_shape = np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    global_shape = tuple(int(v) for v in GLOBAL_INPUT_SHAPE)
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            if partition == 'fold0_fit':
                expected = len(component_records.get(subject_id, []))
                if int(count) != int(expected):
                    raise RuntimeError(
                        f'{subject_id}: resolution audit reconstructed '
                        f'{count} components; frozen audit has {expected}.'
                    )
            pad_start = (canvas_shape - np.asarray(support.shape)) // 2
            cell_sets = []
            component_metadata = []
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for component_id in range(1, int(count) + 1):
                coordinates = np.argwhere(labels == component_id)
                pooled = np.unique(
                    (coordinates + pad_start) // downsample, axis=0
                )
                cell_ids = set(
                    np.ravel_multi_index(pooled.T, global_shape).tolist()
                )
                cell_sets.append(cell_ids)
                voxels = int(len(coordinates))
                volume_ml = voxels * voxel_volume_ml
                component_metadata.append((voxels, volume_ml, pooled))
            cell_frequency = defaultdict(int)
            signature_frequency = defaultdict(int)
            for cells in cell_sets:
                for cell_id in cells:
                    cell_frequency[cell_id] += 1
                signature_frequency[tuple(sorted(cells))] += 1
            matched = r7d_maximum_component_cell_matching(cell_sets)
            collision_components = 0
            duplicate_signature_components = 0
            for component_index, cells in enumerate(cell_sets):
                voxels, volume_ml, pooled = component_metadata[component_index]
                shared_cells = sum(
                    int(cell_frequency[cell_id] > 1) for cell_id in cells
                )
                duplicate_signature = (
                    signature_frequency[tuple(sorted(cells))] > 1
                )
                collision = shared_cells > 0
                collision_components += int(collision)
                duplicate_signature_components += int(duplicate_signature)
                component_rows.append({
                    'development_partition': partition,
                    'subject_id': subject_id,
                    'component_id': int(component_index + 1),
                    'component_tier': pilot_component_tier(volume_ml, voxels),
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'global_proposal_voxels': int(len(cells)),
                    'single_global_voxel': bool(len(cells) == 1),
                    'shared_global_voxels': int(shared_cells),
                    'shares_any_global_voxel': bool(collision),
                    'duplicate_pooled_signature': bool(duplicate_signature),
                    'native_voxels_per_global_voxel': float(
                        voxels / max(len(cells), 1)
                    ),
                })
            subject_rows.append({
                'development_partition': partition,
                'subject_id': subject_id,
                'components': int(count),
                'distinct_global_voxels_used': int(len(cell_frequency)),
                'components_sharing_any_global_voxel': int(collision_components),
                'components_with_duplicate_pooled_signature': int(
                    duplicate_signature_components
                ),
                'maximum_distinct_component_assignment': int(matched),
                'distinct_assignment_ceiling_fraction': float(
                    matched / max(int(count), 1)
                ) if count else 1.0,
            })
            if case_index in {1, len(frame)} or case_index % 20 == 0:
                print(
                    f'  Resolution {partition} {case_index:02d}/{len(frame)}: '
                    f'{subject_id}, components={count}'
                )
    components = pd.DataFrame(component_rows)
    subjects = pd.DataFrame(subject_rows)
    summary_rows = []
    for partition in ['fold0_fit', 'fold0_heldout']:
        for tier in COMPONENT_TIERS:
            group = components[
                (components['development_partition'] == partition)
                & (components['component_tier'] == tier)
            ]
            summary_rows.append({
                'development_partition': partition,
                'component_tier': tier,
                'components': int(len(group)),
                'single_global_voxel_fraction': float(
                    group['single_global_voxel'].mean()
                ) if len(group) else 0.0,
                'global_voxel_collision_fraction': float(
                    group['shares_any_global_voxel'].mean()
                ) if len(group) else 0.0,
                'duplicate_pooled_signature_fraction': float(
                    group['duplicate_pooled_signature'].mean()
                ) if len(group) else 0.0,
                'median_global_proposal_voxels': float(
                    group['global_proposal_voxels'].median()
                ) if len(group) else 0.0,
                'mean_distinct_assignment_ceiling_fraction': float('nan'),
            })
    for partition, group in subjects.groupby('development_partition'):
        summary_rows.append({
            'development_partition': str(partition),
            'component_tier': 'subject_level_distinct_assignment_ceiling',
            'components': int(group['components'].sum()),
            'single_global_voxel_fraction': float('nan'),
            'global_voxel_collision_fraction': float('nan'),
            'duplicate_pooled_signature_fraction': float('nan'),
            'median_global_proposal_voxels': float('nan'),
            'mean_distinct_assignment_ceiling_fraction': float(
                group['distinct_assignment_ceiling_fraction'].mean()
            ),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(components, R7D_RESOLUTION_PATH)
    atomic_write_csv(summary, R7D_RESOLUTION_SUMMARY_PATH)
    return components, subjects, summary


def r7d_validation_artifacts(epoch):
    root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    paths = {
        'case': root / 'case_metrics.csv',
        'component': root / 'component_metrics.csv',
        'tier': root / 'component_tier_metrics.csv',
        'summary': root / 'validation_summary.json',
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise RuntimeError(
            'Completed validation artifacts are missing: ' + ', '.join(missing)
        )
    return paths


def r7d_paired_completed_metrics(r7_best_epoch):
    r5_paths = r7d_validation_artifacts(0)
    r7_paths = r7d_validation_artifacts(r7_best_epoch)
    r5_case = pd.read_csv(r5_paths['case'])
    r7_case = pd.read_csv(r7_paths['case'])
    case_keys = ['subject_id']
    paired_case = r5_case.merge(
        r7_case, on=case_keys, how='outer', suffixes=('_r5', '_r7'),
        validate='one_to_one', indicator=True,
    )
    if not bool((paired_case['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 case metrics do not pair one-to-one.')
    paired_case = paired_case.drop(columns=['_merge'])
    case_delta_columns = [
        'dice', 'voxel_precision', 'voxel_recall',
        'predicted_volume_ml_from_mask', 'predicted_volume_ml_from_router',
        'presence_probability', 'proposal_voxel_precision',
        'proposal_voxel_recall', 'proposal_predicted_to_target_volume_ratio',
    ]
    for name in case_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_case and right in paired_case:
            paired_case[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_case[right], errors='coerce')
                - pd.to_numeric(paired_case[left], errors='coerce')
            )

    r5_component = pd.read_csv(r5_paths['component'])
    r7_component = pd.read_csv(r7_paths['component'])
    component_keys = ['subject_id', 'component_id']
    paired_component = r5_component.merge(
        r7_component, on=component_keys, how='outer',
        suffixes=('_r5', '_r7'), validate='one_to_one', indicator=True,
    )
    if not bool((paired_component['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 component metrics do not pair one-to-one.')
    paired_component = paired_component.drop(columns=['_merge'])
    component_delta_columns = [
        'final_probability_maximum', 'proposal_probability_maximum',
        'final_maximum_component_iou', 'proposal_maximum_component_iou',
        'proposal_minimum_distance_mm',
        'budgeted_proposal_maximum_component_iou',
        'budgeted_proposal_minimum_distance_mm',
    ]
    for name in component_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_component[right], errors='coerce')
                - pd.to_numeric(paired_component[left], errors='coerce')
            )
    for name in [
        'final_detected', 'proposal_covered',
        'budgeted_proposal_within_8mm',
        'representation_failure_below_0p10',
    ]:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                paired_component[right].map(as_bool).astype(int)
                - paired_component[left].map(as_bool).astype(int)
            )
    atomic_write_csv(paired_case, R7D_PAIRED_CASE_PATH)
    atomic_write_csv(paired_component, R7D_PAIRED_COMPONENT_PATH)

    candidates = []
    positive_case_candidates = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    for label, ascending in [
        ('largest_case_dice_regression', True),
        ('largest_case_dice_improvement', False),
    ]:
        ranked = positive_case_candidates.sort_values(
            'dice_delta_r7_minus_r5', ascending=ascending
        ).head(5)
        for rank, item in enumerate(ranked.to_dict('records'), start=1):
            candidates.append({
                'candidate_reason': label,
                'rank': rank,
                'subject_id': str(item['subject_id']),
                'component_id': np.nan,
                'component_tier': '',
                'r5_value': float(item['dice_r5']),
                'r7_value': float(item['dice_r7']),
                'delta_r7_minus_r5': float(
                    item['dice_delta_r7_minus_r5']
                ),
            })
    small = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ].copy()
    small = small.sort_values(
        'proposal_probability_maximum_delta_r7_minus_r5', ascending=True
    ).head(10)
    for rank, item in enumerate(small.to_dict('records'), start=1):
        candidates.append({
            'candidate_reason': 'largest_small_proposal_probability_regression',
            'rank': rank,
            'subject_id': str(item['subject_id']),
            'component_id': int(item['component_id']),
            'component_tier': str(item['component_tier_r7']),
            'r5_value': float(item['proposal_probability_maximum_r5']),
            'r7_value': float(item['proposal_probability_maximum_r7']),
            'delta_r7_minus_r5': float(
                item['proposal_probability_maximum_delta_r7_minus_r5']
            ),
        })
    candidate_frame = pd.DataFrame(candidates)
    atomic_write_csv(candidate_frame, R7D_VISUAL_CANDIDATES_PATH)
    return paired_case, paired_component, candidate_frame


def r7d_checkpoint_pareto(r7_best_epoch):
    history = pd.read_csv(PILOT_HISTORY_PATH).sort_values('epoch').copy()
    maximize = [
        'validation_mean_dice',
        'validation_global_dice',
        'validation_small_mean_dice',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'proposal_voxel_average_precision_256bin',
    ]
    maximize = [name for name in maximize if name in history.columns]
    values = history[maximize].apply(pd.to_numeric, errors='coerce').fillna(
        -np.inf
    ).to_numpy(dtype=np.float64)
    pareto = np.ones(len(history), dtype=bool)
    for index in range(len(history)):
        dominated = np.any(
            np.all(values >= values[index], axis=1)
            & np.any(values > values[index], axis=1)
        )
        pareto[index] = not bool(dominated)
    history['pareto_nondominated'] = pareto
    history['is_selected_best_epoch'] = (
        history['epoch'].astype(int) == int(r7_best_epoch)
    )
    best_rows = {}
    metric_flags = {}
    for name in maximize + ['pilot_checkpoint_score']:
        if name not in history:
            continue
        values_series = pd.to_numeric(history[name], errors='coerce')
        best_index = values_series.idxmax()
        best_epoch = int(history.loc[best_index, 'epoch'])
        best_value = float(values_series.loc[best_index])
        flag = f'is_best_{name}'
        history[flag] = history['epoch'].astype(int) == best_epoch
        best_rows[name] = {'epoch': best_epoch, 'value': best_value}
        metric_flags[name] = flag
    atomic_write_csv(history, R7D_CHECKPOINT_PARETO_PATH)
    return history, best_rows


def r7d_branch_comparison(r5_best_epoch, r7_best_epoch):
    sources = [
        ('r5', R5_PILOT_ROOT / 'branch_reliance_history.csv', r5_best_epoch),
        ('r7', PILOT_BRANCH_HISTORY_PATH, r7_best_epoch),
    ]
    selected_frames = []
    for checkpoint, path, epoch in sources:
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        available_epochs = sorted(frame['epoch'].astype(int).unique().tolist())
        if not available_epochs:
            continue
        selected_epoch = (
            int(epoch) if int(epoch) in available_epochs
            else min(available_epochs, key=lambda value: abs(value - int(epoch)))
        )
        selected = frame[frame['epoch'].astype(int) == selected_epoch].copy()
        selected['checkpoint'] = checkpoint
        selected['requested_best_epoch'] = int(epoch)
        selected['selected_probe_epoch'] = int(selected_epoch)
        selected['exact_best_epoch_probe'] = bool(selected_epoch == int(epoch))
        selected_frames.append(selected)
    if not selected_frames:
        comparison = pd.DataFrame(columns=[
            'checkpoint', 'ablation', 'mean_target_probability_delta'
        ])
    else:
        raw = pd.concat(selected_frames, ignore_index=True)
        rows = []
        for (checkpoint, ablation), group in raw.groupby(
            ['checkpoint', 'ablation']
        ):
            rows.append({
                'checkpoint': str(checkpoint),
                'ablation': str(ablation),
                'subjects': int(group['subject_id'].nunique()),
                'requested_best_epoch': int(
                    group['requested_best_epoch'].iloc[0]
                ),
                'selected_probe_epoch': int(
                    group['selected_probe_epoch'].iloc[0]
                ),
                'exact_best_epoch_probe': bool(
                    group['exact_best_epoch_probe'].iloc[0]
                ),
                'mean_target_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_target_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_local_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_local_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_proposal_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_proposal_probability_delta'],
                    errors='coerce',
                ).mean()),
            })
        comparison = pd.DataFrame(rows)
        if set(comparison['checkpoint']) == {'r5', 'r7'}:
            pivot = comparison.pivot(
                index='ablation', columns='checkpoint',
                values='mean_target_probability_delta',
            )
            deltas = (pivot['r7'] - pivot['r5']).to_dict()
            comparison['target_delta_change_r7_minus_r5'] = comparison[
                'ablation'
            ].map(deltas)
    atomic_write_csv(comparison, R7D_BRANCH_PATH)
    return comparison


def r7d_global_component_masks(labels, count, pad_start):
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pad = np.asarray(pad_start, dtype=np.int64)
    all_coordinates = np.argwhere(labels > 0)
    full = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    if len(all_coordinates):
        pooled = np.unique((all_coordinates + pad) // downsample, axis=0)
        full[tuple(pooled.T)] = True
    components = []
    for component_id in range(1, int(count) + 1):
        coordinates = np.argwhere(labels == component_id)
        pooled = np.unique((coordinates + pad) // downsample, axis=0)
        components.append(pooled)
    return full, components


def r7d_global_domain(native_shape, pad_start):
    start = np.asarray(pad_start, dtype=np.int64)
    end = start + np.asarray(native_shape, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    low_start = start // downsample
    low_end = (end + downsample - 1) // downsample
    domain = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    slices = tuple(
        slice(int(low_start[axis]), int(low_end[axis])) for axis in range(3)
    )
    domain[slices] = True
    return domain


def r7d_component_logit_margins(
    logits, full_target, component_coordinates, domain, subject_id,
    partition, checkpoint, spacing,
):
    rows = []
    outer_radius = np.asarray(R7_OUTER_STRUCTURE.shape, dtype=np.int64) // 2
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    native_voxel_volume = voxel_volume_ml
    for component_id, pooled in enumerate(component_coordinates, start=1):
        if len(pooled) == 0:
            continue
        minimum = np.maximum(pooled.min(axis=0) - outer_radius, 0)
        maximum = np.minimum(
            pooled.max(axis=0) + outer_radius + 1,
            np.asarray(GLOBAL_INPUT_SHAPE),
        )
        slices = tuple(
            slice(int(minimum[axis]), int(maximum[axis]))
            for axis in range(3)
        )
        exact = np.zeros(tuple((maximum - minimum).tolist()), dtype=bool)
        local_coordinates = pooled - minimum
        exact[tuple(local_coordinates.T)] = True
        _, _, negative = r7_component_regions(
            exact, full_target[slices], domain[slices]
        )
        positive_logits = logits[slices][exact]
        negative_logits = logits[slices][negative]
        hard_count = min(
            len(negative_logits),
            max(R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * len(positive_logits)),
        )
        if hard_count > 0:
            hard_negative = np.partition(
                negative_logits, len(negative_logits) - hard_count
            )[-hard_count:]
            hard_negative_mean = float(hard_negative.mean())
            hard_negative_maximum = float(hard_negative.max())
        else:
            hard_negative_mean = float('nan')
            hard_negative_maximum = float('nan')
        native_voxels_estimate = int(len(pooled) * np.prod(GLOBAL_DOWNSAMPLE))
        volume_ml_estimate = native_voxels_estimate * native_voxel_volume
        positive_mean = float(positive_logits.mean())
        positive_minimum = float(positive_logits.min())
        positive_maximum = float(positive_logits.max())
        rows.append({
            'checkpoint': checkpoint,
            'development_partition': partition,
            'subject_id': subject_id,
            'component_id': int(component_id),
            'component_tier_from_pooled_extent': pilot_component_tier(
                volume_ml_estimate, native_voxels_estimate
            ),
            'global_proposal_voxels': int(len(pooled)),
            'positive_logit_mean': positive_mean,
            'positive_logit_minimum': positive_minimum,
            'positive_logit_maximum': positive_maximum,
            'hard_negative_logit_mean': hard_negative_mean,
            'hard_negative_logit_maximum': hard_negative_maximum,
            'positive_minus_hard_negative_mean_margin': (
                positive_mean - hard_negative_mean
                if np.isfinite(hard_negative_mean) else float('nan')
            ),
            'positive_minimum_minus_hard_negative_maximum_margin': (
                positive_minimum - hard_negative_maximum
                if np.isfinite(hard_negative_maximum) else float('nan')
            ),
            'positive_probability_mean': float(
                1.0 / (1.0 + np.exp(-np.clip(positive_mean, -30, 30)))
            ),
            'negative_annulus_voxels': int(negative.sum()),
            'hard_negative_voxels_used': int(hard_count),
        })
    return rows


def r7d_topk_prediction(probability, budget):
    probability = np.asarray(probability, dtype=np.float32)
    budget = int(np.clip(int(budget), 1, probability.size))
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape)


def r7d_budget_metrics(
    probability, target, spacing, predicted_volume_ml, subject_id,
    checkpoint,
):
    target = np.asarray(target, dtype=bool)
    target_voxels = int(target.sum())
    if target_voxels == 0:
        return []
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    labels, count = ndimage.label(
        target, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    component_objects = ndimage.find_objects(labels)
    component_metadata = []
    for component_id, component_slice in enumerate(component_objects, start=1):
        if component_slice is None:
            continue
        local = labels[component_slice] == component_id
        voxels = int(local.sum())
        volume_ml = voxels * voxel_volume_ml
        component_metadata.append((
            component_id, component_slice, local,
            pilot_component_tier(volume_ml, voxels),
        ))
    budgets = [
        ('router_x1', max(1, int(math.ceil(predicted_volume_ml / voxel_volume_ml)))),
        ('router_x2', max(1, int(math.ceil(2.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('router_x3', max(1, int(math.ceil(3.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('oracle_target_x3', max(1, int(math.ceil(3.0 * target_voxels)))),
    ]
    rows = []
    for budget_name, requested_budget in budgets:
        prediction = r7d_topk_prediction(probability, requested_budget)
        predicted_voxels = int(prediction.sum())
        intersection = int((prediction & target).sum())
        distance = ndimage.distance_transform_edt(
            ~prediction, sampling=spacing
        )
        within_8 = []
        small_within_8 = []
        for _, component_slice, component_local, tier in component_metadata:
            minimum_distance = float(
                distance[component_slice][component_local].min(initial=np.inf)
            )
            covered = bool(minimum_distance <= 8.0)
            within_8.append(covered)
            if tier in R7_SMALL_COMPONENT_TIERS:
                small_within_8.append(covered)
        rows.append({
            'checkpoint': checkpoint,
            'subject_id': subject_id,
            'budget_policy': budget_name,
            'router_predicted_volume_ml': float(predicted_volume_ml),
            'target_volume_ml': float(target_voxels * voxel_volume_ml),
            'router_to_target_volume_ratio': float(
                predicted_volume_ml / max(target_voxels * voxel_volume_ml, 1e-8)
            ),
            'requested_budget_voxels': int(requested_budget),
            'selected_budget_voxels': predicted_voxels,
            'selected_to_target_voxel_ratio': float(
                predicted_voxels / max(target_voxels, 1)
            ),
            'voxel_precision': float(intersection / max(predicted_voxels, 1)),
            'voxel_recall': float(intersection / max(target_voxels, 1)),
            'dice': float(
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            ),
            'components': int(len(component_metadata)),
            'component_within_8mm_fraction': float(np.mean(within_8)),
            'small_components': int(len(small_within_8)),
            'small_component_within_8mm_fraction': float(
                np.mean(small_within_8)
            ) if small_within_8 else float('nan'),
        })
    return rows


def r7d_load_model(checkpoint_path, expected_version, expected_revision, device):
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=False
    )
    checks = {
        'pilot_version': checkpoint.get('pilot_version') == expected_version,
        'code_revision': checkpoint.get('code_revision') == expected_revision,
        'parameter_count': int(checkpoint.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS,
        'ema_model_state': 'ema_model_state' in checkpoint,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'Unsafe read-only checkpoint {checkpoint_path}: '
            + ', '.join(failed)
        )
    model = VascuFateNetV22(pilot_schema).to(device)
    model.load_state_dict(checkpoint['ema_model_state'], strict=True)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    del checkpoint
    return model


@torch.no_grad()
def r7d_checkpoint_margin_and_volume_audit(
    checkpoint_label, checkpoint_path, expected_version,
    expected_revision, device,
):
    model = r7d_load_model(
        checkpoint_path, expected_version, expected_revision, device
    )
    margin_rows = []
    budget_rows = []
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            case = pilot_load_validation_case(row)
            batch = pilot_stack_sample(case['base_sample'], device)
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(model, batch)
            proposal_logits_tensor = cache['proposal_logits'].float()
            logits = proposal_logits_tensor[0, 0].cpu().numpy()
            predicted_volume_ml = float(torch.expm1(
                cache['routed']['log_volume_prediction'].float()
            )[0].clamp_min(0).cpu())
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            full_global, component_coordinates = r7d_global_component_masks(
                labels, count, case['pad_start']
            )
            domain = r7d_global_domain(case['native_shape'], case['pad_start'])
            rows = r7d_component_logit_margins(
                logits, full_global, component_coordinates, domain,
                subject_id, partition, checkpoint_label, spacing,
            )
            
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for item in rows:
                component_id = int(item['component_id'])
                voxels = int((labels == component_id).sum())
                volume_ml = voxels * voxel_volume_ml
                item['component_voxels'] = voxels
                item['component_volume_ml'] = volume_ml
                item['component_tier'] = pilot_component_tier(
                    volume_ml, voxels
                )
            margin_rows.extend(rows)

            if (
                partition == 'fold0_heldout'
                and as_bool(row['trial8_hard_metric_eligible'])
                and bool(case['target'].any())
            ):
                proposal_canvas = F.interpolate(
                    torch.sigmoid(proposal_logits_tensor),
                    size=GLOBAL_CANVAS_SHAPE,
                    mode='trilinear', align_corners=False,
                )[0, 0].cpu().numpy()
                native_slices = tuple(
                    slice(
                        case['pad_start'][axis],
                        case['pad_start'][axis] + case['native_shape'][axis],
                    )
                    for axis in range(3)
                )
                proposal_native = proposal_canvas[native_slices]
                budget_rows.extend(r7d_budget_metrics(
                    proposal_native, case['target'], case['spacing'],
                    predicted_volume_ml, subject_id, checkpoint_label,
                ))
            del batch, cache, proposal_logits_tensor, case
            if case_index in {1, len(frame)} or case_index % 10 == 0:
                print(
                    f'  {checkpoint_label} {partition} '
                    f'{case_index:02d}/{len(frame)}: {subject_id}'
                )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return margin_rows, budget_rows


def r7d_margin_summary(frame):
    rows = []
    for (checkpoint, partition, tier), group in frame.groupby([
        'checkpoint', 'development_partition', 'component_tier'
    ]):
        margin = pd.to_numeric(
            group['positive_minus_hard_negative_mean_margin'],
            errors='coerce',
        )
        rows.append({
            'checkpoint': str(checkpoint),
            'development_partition': str(partition),
            'component_tier': str(tier),
            'components': int(len(group)),
            'mean_logit_margin': float(margin.mean()),
            'median_logit_margin': float(margin.median()),
            'positive_margin_fraction': float((margin > 0).mean()),
            'mean_positive_probability': float(pd.to_numeric(
                group['positive_probability_mean'], errors='coerce'
            ).mean()),
        })
    return pd.DataFrame(rows)


def r7d_volume_budget_summary(frame):
    rows = []
    for (checkpoint, policy), group in frame.groupby(
        ['checkpoint', 'budget_policy']
    ):
        rows.append({
            'checkpoint': str(checkpoint),
            'budget_policy': str(policy),
            'subjects': int(group['subject_id'].nunique()),
            'mean_dice': float(group['dice'].mean()),
            'mean_voxel_precision': float(group['voxel_precision'].mean()),
            'mean_voxel_recall': float(group['voxel_recall'].mean()),
            'mean_component_within_8mm_fraction': float(
                group['component_within_8mm_fraction'].mean()
            ),
            'mean_small_component_within_8mm_fraction': float(
                group['small_component_within_8mm_fraction'].mean()
            ),
            'median_router_to_target_volume_ratio': float(
                group['router_to_target_volume_ratio'].median()
            ),
            'mean_selected_to_target_voxel_ratio': float(
                group['selected_to_target_voxel_ratio'].mean()
            ),
        })
    return pd.DataFrame(rows)


def r7d_small_margin_delta(summary, partition):
    subset = summary[
        (summary['development_partition'] == partition)
        & summary['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    if not len(subset):
        return float('nan')
    weighted = {}
    for checkpoint in ['r5', 'r7']:
        group = subset[subset['checkpoint'] == checkpoint]
        if not len(group):
            weighted[checkpoint] = float('nan')
            continue
        weighted[checkpoint] = float(np.average(
            group['mean_logit_margin'], weights=group['components']
        ))
    return weighted['r7'] - weighted['r5']


def run_r7_failure_mechanism_diagnostic():
    if r7d_existing_result_is_complete():
        gate = load_json(R7_FAILURE_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC ALREADY COMPLETE')
        print('=' * 80)
        print(f'Dominant mechanism: {gate["dominant_failure_mechanism"]}')
        print(f'Diagnostic gate:    {R7_FAILURE_GATE_PATH}')
        print('No source checkpoint or completed diagnostic was rewritten.')
        return

    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r7_best_epoch = int(r7_result['best_epoch'])
    r5_best_epoch = int(r5_result['best_epoch'])
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_sha_before = file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    source_paths = [
        R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH,
        R5_RESULT_GATE_PATH, PILOT_RESULT_GATE_PATH, PILOT_HISTORY_PATH,
    ]
    source_hashes_before = {
        str(path): file_sha256(path) for path in source_paths
    }
    initial_checks = {
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r7_pilot_complete': r7_result.get('pilot_complete') is True,
        'r5_checkpoint_hash_matches_result_gate': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'r7_checkpoint_hash_matches_result_gate': (
            r7_result.get('best_checkpoint_sha256') == r7_sha_before
        ),
        'r7_directional_success_was_false': (
            r7_result.get('r7_directional_success_passed') is False
        ),
        'full_fivefold_training_locked': (
            r7_result.get('full_fivefold_training_authorized') is False
        ),
        'fold0_subject_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
        ),
        'sealed_test_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
        ),
    }
    if not all(initial_checks.values()):
        failed = [name for name, passed in initial_checks.items() if not passed]
        raise RuntimeError(
            'R7 failure diagnostic upstream checks failed: '
            + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC STARTING')
    print('=' * 80)
    print(f'R5 / r7 best epochs:               {r5_best_epoch} / {r7_best_epoch}')
    print('Target access:                     fold-0 development only')
    print('Locked validation targets read:    False')
    print('Test targets read:                 False')
    print('Optimizer / backward / steps:      0 / 0 / 0')

    target_access_frame = pd.DataFrame([
        {
            'subject_id': str(row['subject_id']),
            'development_partition': partition,
            'target_arrays_authorized': True,
            'target_arrays_read_by_diagnostic': True,
            'locked_validation_target_read': False,
            'sealed_test_target_read': False,
        }
        for partition, frame in [
            ('fold0_fit', fold0_fit_df),
            ('fold0_heldout', fold0_heldout_df),
        ]
        for row in frame.to_dict('records')
    ])
    atomic_write_csv(target_access_frame, R7D_TARGET_ACCESS_PATH)

    exposure, exposure_summary = r7d_component_schedule_exposure()
    resolution, resolution_subjects, resolution_summary = (
        r7d_resolution_audit()
    )
    paired_case, paired_component, visual_candidates = (
        r7d_paired_completed_metrics(r7_best_epoch)
    )
    checkpoint_pareto, metric_best_rows = r7d_checkpoint_pareto(
        r7_best_epoch
    )
    branch_comparison = r7d_branch_comparison(
        r5_best_epoch, r7_best_epoch
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            'The train-versus-held-out r5/r7 margin audit requires CUDA.'
        )
    set_determinism(PILOT_SEED)
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    all_margin_rows = []
    all_budget_rows = []
    checkpoint_specs = [
        ('r5', R5_BEST_CHECKPOINT_PATH,
         R5_PILOT_VERSION, R5_PILOT_CODE_REVISION),
        ('r7', PILOT_BEST_CHECKPOINT_PATH,
         PILOT_VERSION, PILOT_CODE_REVISION),
    ]
    for label, path, version, revision in checkpoint_specs:
        print(f'\nRead-only global proposal audit: {label}')
        margin_rows, budget_rows = r7d_checkpoint_margin_and_volume_audit(
            label, path, version, revision, device
        )
        all_margin_rows.extend(margin_rows)
        all_budget_rows.extend(budget_rows)
    margins = pd.DataFrame(all_margin_rows)
    budgets = pd.DataFrame(all_budget_rows)
    margin_summary = r7d_margin_summary(margins)
    budget_summary = r7d_volume_budget_summary(budgets)
    atomic_write_csv(margins, R7D_MARGIN_PATH)
    atomic_write_csv(margin_summary, R7D_MARGIN_SUMMARY_PATH)
    atomic_write_csv(budgets, R7D_VOLUME_BUDGET_PATH)
    atomic_write_csv(budget_summary, R7D_VOLUME_BUDGET_SUMMARY_PATH)

    small_exposure = exposure[
        exposure['local_contrastive_eligible'].astype(bool)
    ]
    never_selected_fraction = float(
        small_exposure['never_selected'].mean()
    )
    heldout_small_resolution = resolution[
        (resolution['development_partition'] == 'fold0_heldout')
        & resolution['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    heldout_small_collision_fraction = float(
        heldout_small_resolution['shares_any_global_voxel'].mean()
    ) if len(heldout_small_resolution) else float('nan')
    heldout_assignment_ceiling = float(
        resolution_subjects[
            resolution_subjects['development_partition'] == 'fold0_heldout'
        ]['distinct_assignment_ceiling_fraction'].mean()
    )
    fit_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_fit'
    )
    heldout_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_heldout'
    )
    positive_paired_cases = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    case_dice_delta = float(
        positive_paired_cases['dice_delta_r7_minus_r5'].mean()
    )
    small_components = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ]
    small_proposal_probability_delta = float(pd.to_numeric(
        small_components[
            'proposal_probability_maximum_delta_r7_minus_r5'
        ], errors='coerce'
    ).mean())
    r7_budget = budget_summary[
        budget_summary['checkpoint'] == 'r7'
    ]
    router_x3 = r7_budget[
        r7_budget['budget_policy'] == 'router_x3'
    ]
    oracle_x3 = r7_budget[
        r7_budget['budget_policy'] == 'oracle_target_x3'
    ]
    router_small_within8 = float(
        router_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    oracle_small_within8 = float(
        oracle_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    deployment_budget_gap = oracle_small_within8 - router_small_within8
    selected_epoch = int(r7_best_epoch)
    mean_dice_best_epoch = int(
        metric_best_rows['validation_mean_dice']['epoch']
    )
    checkpoint_selection_mismatch = selected_epoch != mean_dice_best_epoch

    mechanism_evidence = {
        'component_exposure_imbalance': bool(never_selected_fraction > 0.10),
        'low_resolution_component_collision': bool(
            heldout_small_collision_fraction > 0.10
            or heldout_assignment_ceiling < 0.95
        ),
        'contrastive_train_heldout_margin_divergence': bool(
            np.isfinite(fit_small_margin_delta)
            and np.isfinite(heldout_small_margin_delta)
            and fit_small_margin_delta > heldout_small_margin_delta + 0.10
        ),
        'small_proposal_signal_regressed': bool(
            small_proposal_probability_delta < 0.0
        ),
        'deployment_volume_router_limits_coverage': bool(
            deployment_budget_gap > 0.10
        ),
        'checkpoint_objective_mismatch': bool(checkpoint_selection_mismatch),
        'r7_mean_dice_regressed': bool(case_dice_delta < 0.0),
        'r7_representation_failure_extreme': bool(
            float(r7_result['best_validation_summary'][
                'small_component_representation_failure_fraction'
            ]) > 0.75
        ),
    }
    if mechanism_evidence['contrastive_train_heldout_margin_divergence']:
        dominant_mechanism = (
            'low-resolution local-contrastive over-specialization with '
            'poor held-out satellite transfer'
        )
    elif mechanism_evidence['component_exposure_imbalance']:
        dominant_mechanism = (
            'incomplete atomic-component exposure in the stochastic curriculum'
        )
    elif mechanism_evidence['low_resolution_component_collision']:
        dominant_mechanism = (
            'global proposal-grid resolution and component-collision ceiling'
        )
    else:
        dominant_mechanism = (
            'proposal supervision/refinement mismatch rather than missing '
            'clinical or arterial routing capacity'
        )
    recommended_r8 = {
        'architecture': (
            'Preserve the r5 V2.2 global/arterial/clinical path; add a bounded '
            'native-resolution local satellite-proposal head and fuse it as '
            'a zero-initialized residual, leaving the coarse global proposal '
            'responsible for burden and broad localization.'
        ),
        'curriculum': (
            'Replace stochastic inverse-size component draws with a '
            'coverage-guaranteed component cycle before repeats, stratified '
            'by subvoxel/micro/tiny/small tier and subject.'
        ),
        'objective': (
            'Apply component contrast at native local resolution, normalize '
            'per component, cap its gradient contribution against the main '
            'segmentation loss, and retain the coarse proposal loss.'
        ),
        'deployment': (
            'Train and calibrate the predicted-volume router on fold-fit data '
            'only; use predicted rather than target volume for any inference '
            'budget and report the oracle budget only as a diagnostic ceiling.'
        ),
        'selection': (
            'Use a Pareto-safe checkpoint rule that requires no mean-Dice '
            'regression and explicit held-out small-component improvement; '
            'do not optimize the present composite score alone.'
        ),
        'next_stage': (
            'Build an R8 read-only architecture/loss preflight first, then one '
            'fold-0 pilot. Five-fold and locked validation/test evaluation '
            'remain unauthorized.'
        ),
    }

    source_hashes_after = {
        str(path): file_sha256(path) for path in source_paths
    }
    final_checks = {
        **initial_checks,
        'fit_component_count_matches_frozen_872': int(
            (resolution['development_partition'] == 'fold0_fit').sum()
        ) == 872,
        'exposure_covers_every_frozen_fit_component': len(exposure)
        == len(component_df),
        'paired_case_metrics_have_20_subjects': len(paired_case) == 20,
        'paired_component_metrics_are_nonempty': len(paired_component) > 0,
        'margin_audit_has_both_checkpoints': set(margins['checkpoint'])
        == {'r5', 'r7'},
        'margin_audit_has_fit_and_heldout': set(
            margins['development_partition']
        ) == {'fold0_fit', 'fold0_heldout'},
        'all_finite_margin_inputs': bool(np.isfinite(pd.to_numeric(
            margins['positive_logit_mean'], errors='coerce'
        )).all()),
        'deployment_budget_uses_heldout_development_only': bool(
            budgets['subject_id'].isin(fold0_heldout_df['subject_id']).all()
        ),
        'development_target_access_has_exactly_102_subjects': (
            len(target_access_frame) == 102
            and target_access_frame['subject_id'].nunique() == 102
        ),
        'development_target_access_excludes_locked_splits': bool(
            (~target_access_frame['locked_validation_target_read']).all()
            and (~target_access_frame['sealed_test_target_read']).all()
        ),
        'r5_checkpoint_unchanged': r5_sha_before
        == file_sha256(R5_BEST_CHECKPOINT_PATH),
        'r7_checkpoint_unchanged': r7_sha_before
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'all_source_artifacts_unchanged': source_hashes_before
        == source_hashes_after,
        'optimizer_not_constructed': True,
        'backward_not_called': True,
        'optimizer_steps_are_zero': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    diagnostic_passed = all(final_checks.values())

    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': R7_FAILURE_DIAGNOSTIC_VERSION,
        'code_revision': R7_FAILURE_DIAGNOSTIC_CODE_REVISION,
        'r5_best_epoch': r5_best_epoch,
        'r7_best_epoch': r7_best_epoch,
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_sha256': r7_sha_before,
        'observed_effects': {
            'mean_paired_case_dice_delta_r7_minus_r5': case_dice_delta,
            'small_proposal_probability_delta_r7_minus_r5': (
                small_proposal_probability_delta
            ),
            'eligible_components_never_selected_fraction': (
                never_selected_fraction
            ),
            'heldout_small_global_voxel_collision_fraction': (
                heldout_small_collision_fraction
            ),
            'heldout_distinct_assignment_ceiling_fraction': (
                heldout_assignment_ceiling
            ),
            'fit_small_logit_margin_delta_r7_minus_r5': (
                fit_small_margin_delta
            ),
            'heldout_small_logit_margin_delta_r7_minus_r5': (
                heldout_small_margin_delta
            ),
            'r7_router_x3_small_within_8mm_fraction': router_small_within8,
            'r7_oracle_x3_small_within_8mm_fraction': oracle_small_within8,
            'deployment_to_oracle_small_within_8mm_gap': (
                deployment_budget_gap
            ),
            'selected_checkpoint_epoch': selected_epoch,
            'mean_dice_best_epoch': mean_dice_best_epoch,
        },
        'mechanism_evidence': mechanism_evidence,
        'dominant_failure_mechanism': dominant_mechanism,
        'recommended_r8_contract': recommended_r8,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': False,
        'backward_called': False,
        'optimizer_steps': 0,
        'r8_preflight_implementation_allowed': bool(diagnostic_passed),
        'r8_training_authorized': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(summary, R7_FAILURE_SUMMARY_PATH)
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(value)}
        for name, value in final_checks.items()
    ])
    atomic_write_csv(checks_frame, R7D_CHECKS_PATH)
    output_hashes = {
        path.name: file_sha256(path) for path in R7D_OUTPUT_PATHS
    }
    gate = {
        **summary,
        'diagnostic_complete': bool(diagnostic_passed),
        'diagnostic_passed': bool(diagnostic_passed),
        'checks': final_checks,
        'source_sha256_before': source_hashes_before,
        'source_sha256_after': source_hashes_after,
        'output_sha256': output_hashes,
        'diagnostic_root': str(R7_FAILURE_DIAGNOSTIC_ROOT),
        'summary_path': str(R7_FAILURE_SUMMARY_PATH),
    }
    atomic_write_json(gate, R7_FAILURE_GATE_PATH)
    if not diagnostic_passed:
        failed = [
            name for name, passed in final_checks.items() if not passed
        ]
        raise RuntimeError(
            'TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC FAILED. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nComponent exposure by tier:')
    display(exposure_summary)
    print('\nProposal-grid resolution summary:')
    display(resolution_summary)
    print('\nTrain-versus-held-out proposal-logit margins:')
    display(margin_summary)
    print('\nDeployment-legal predicted-volume budget audit:')
    display(budget_summary)
    print('\nBranch reliance comparison:')
    display(branch_comparison)
    print('\nDiagnostic checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Dominant mechanism:                 {dominant_mechanism}')
    print(f'Eligible components never selected: {never_selected_fraction:.4f}')
    print(f'Fit small-margin change r7-r5:      {fit_small_margin_delta:+.4f}')
    print(f'Heldout small-margin change r7-r5:  {heldout_small_margin_delta:+.4f}')
    print(f'R7 router-x3 small within 8 mm:     {router_small_within8:.4f}')
    print(f'R7 oracle-x3 small within 8 mm:     {oracle_small_within8:.4f}')
    print(f'Mean-Dice best / selected epoch:    {mean_dice_best_epoch} / {selected_epoch}')
    print('Optimizer / backward / steps:       0 / 0 / 0')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('R8 preflight implementation allowed: True')
    print('R8 training authorized:             False')
    print('Full five-fold training authorized: False')
    print(f'Diagnostic summary:                 {R7_FAILURE_SUMMARY_PATH}')
    print(f'Diagnostic gate:                    {R7_FAILURE_GATE_PATH}')
    print(
        '\nNext: construct the read-only R8 dual-resolution satellite-proposal '
        'and component-cycled curriculum preflight; do not train yet.'
    )


run_r7_failure_mechanism_diagnostic()






R8_MODEL_VERSION = 'trial8_vascufate_net_v2_3_dual_resolution_satellite'
R8_PREFLIGHT_VERSION = 'v1_r5_warmstart_component_cycle_native_satellite'
R8_CODE_REVISION = (
    'r1_native_component_absolute_calibration_pareto_guard'
)
R8_SEED = 20261007
R8_SATELLITE_GATE_BOUND = 0.20
R8_SATELLITE_FEATURE_CHANNELS = 16
R8_COMPONENT_POSITIVE_PATCHES = 112
R8_TOPOLOGY_NEGATIVE_PATCHES = 28
R8_BRAIN_NEGATIVE_PATCHES = 28
R8_SCHEDULE_EPOCHS = 40
R8_NATIVE_INNER_RADIUS_MM = 2.0
R8_NATIVE_OUTER_RADIUS_MM = 12.0
R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR = 1.0
R8_MAXIMUM_NEW_PARAMETERS = 250_000
R8_TIER_QUOTAS = {
    'subvoxel_or_single_voxel': 2,
    'micro': 76,
    'tiny': 27,
    'small': 7,
}

R8_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / R8_MODEL_VERSION / R8_PREFLIGHT_VERSION
)
R8_PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
R8_ARCHITECTURE_PATH = R8_PREFLIGHT_ROOT / 'r8_architecture_config.json'
R8_TRAINING_POLICY_PATH = R8_PREFLIGHT_ROOT / 'r8_training_policy.json'
R8_SCHEDULE_PATH = R8_PREFLIGHT_ROOT / 'r8_component_cycle_schedule.csv'
R8_SCHEDULE_SUMMARY_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_component_cycle_summary.csv'
)
R8_NATIVE_TARGET_AUDIT_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_native_component_target_audit.csv'
)
R8_PARAMETER_PATH = R8_PREFLIGHT_ROOT / 'r8_parameter_audit.csv'
R8_OUTPUT_PATH = R8_PREFLIGHT_ROOT / 'r8_output_shape_audit.csv'
R8_NEUTRALITY_PATH = R8_PREFLIGHT_ROOT / 'r8_neutrality_audit.csv'
R8_GRADIENT_PATH = R8_PREFLIGHT_ROOT / 'r8_gradient_audit.csv'
R8_LOSS_PATH = R8_PREFLIGHT_ROOT / 'r8_loss_smoke_audit.csv'
R8_CHECKS_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_checks.csv'
R8_GATE_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_gate.json'

R8_OUTPUT_ARTIFACTS = [
    R8_ARCHITECTURE_PATH,
    R8_TRAINING_POLICY_PATH,
    R8_SCHEDULE_PATH,
    R8_SCHEDULE_SUMMARY_PATH,
    R8_NATIVE_TARGET_AUDIT_PATH,
    R8_PARAMETER_PATH,
    R8_OUTPUT_PATH,
    R8_NEUTRALITY_PATH,
    R8_GRADIENT_PATH,
    R8_LOSS_PATH,
    R8_CHECKS_PATH,
]


def r8_existing_preflight_complete():
    if not R8_GATE_PATH.exists():
        return False
    gate = load_json(R8_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('model_version') == R8_MODEL_VERSION
        and gate.get('preflight_version') == R8_PREFLIGHT_VERSION
        and gate.get('code_revision') == R8_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_failure_diagnostic_sha256')
        == file_sha256(R7_FAILURE_GATE_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R8_OUTPUT_ARTIFACTS
    )


def r8_tier_records():
    records = {}
    for tier in R8_TIER_QUOTAS:
        frame = component_df[
            component_df['component_tier'].astype(str) == tier
        ].sort_values(['subject_id', 'atomic_component_id'])
        records[tier] = frame.to_dict('records')
        if not records[tier]:
            raise RuntimeError(f'R8 component tier {tier!r} is empty.')
    return records


R8_COMPONENT_RECORDS_BY_TIER = r8_tier_records()


def r8_tier_permutation(tier, cycle_index):
    tier_index = list(R8_TIER_QUOTAS).index(str(tier))
    rng = np.random.default_rng(
        R8_SEED + tier_index * 10_000_019 + int(cycle_index) * 1_000_003
    )
    count = len(R8_COMPONENT_RECORDS_BY_TIER[str(tier)])
    return rng.permutation(count).tolist()


def r8_component_for_position(tier, position):
    records = R8_COMPONENT_RECORDS_BY_TIER[str(tier)]
    cycle_index = int(position) // len(records)
    offset = int(position) % len(records)
    permutation = r8_tier_permutation(tier, cycle_index)
    item = dict(records[int(permutation[offset])])
    item['r8_tier_cycle_index'] = int(cycle_index)
    item['r8_tier_cycle_offset'] = int(offset)
    return item


def r8_build_epoch_schedule(epoch):
    epoch = int(epoch)
    if epoch < 1:
        raise ValueError('R8 epoch must be positive.')
    positives = []
    for tier, quota in R8_TIER_QUOTAS.items():
        for slot in range(int(quota)):
            position = (epoch - 1) * int(quota) + slot
            component = r8_component_for_position(tier, position)
            subject_id = str(component['subject_id'])
            jitter_rng = np.random.default_rng(
                R8_SEED + epoch * 10_000_019
                + list(R8_TIER_QUOTAS).index(tier) * 104_729 + slot
            )
            center = np.asarray([
                component['centroid_x_vox'],
                component['centroid_y_vox'],
                component['centroid_z_vox'],
            ], dtype=np.float64)
            jitter = np.asarray([
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-3, 4),
            ])
            shape = np.asarray(
                pilot_mask_records[subject_id]['shape'], dtype=np.int64
            )
            center = np.clip(np.rint(center).astype(np.int64) + jitter, 0, shape - 1)
            positives.append({
                'epoch': epoch,
                'sample_type': 'positive',
                'subject_id': subject_id,
                'center_native': tuple(int(value) for value in center),
                'atomic_component_id': int(
                    component['atomic_component_id']
                ),
                'component_tier': tier,
                'component_volume_ml': float(
                    component['component_volume_ml']
                ),
                'r8_tier_cycle_index': int(
                    component['r8_tier_cycle_index']
                ),
                'r8_tier_cycle_offset': int(
                    component['r8_tier_cycle_offset']
                ),
            })
    if len(positives) != R8_COMPONENT_POSITIVE_PATCHES:
        raise RuntimeError(
            f'R8 epoch {epoch} has {len(positives)} positive patches.'
        )
    inherited = pilot_build_epoch_schedule(epoch)
    negatives = [
        dict(item) for item in inherited
        if str(item['sample_type']) != 'positive'
    ]
    if sum(str(item['sample_type']) == 'topology_negative' for item in negatives) != 28:
        raise RuntimeError('R8 topology-negative count changed.')
    if sum(str(item['sample_type']) == 'brain_negative' for item in negatives) != 28:
        raise RuntimeError('R8 brain-negative count changed.')
    for item in negatives:
        item['epoch'] = epoch
        item['r8_tier_cycle_index'] = -1
        item['r8_tier_cycle_offset'] = -1
    schedule = positives + negatives
    rng = np.random.default_rng(R8_SEED + epoch * 1_299_721)
    rng.shuffle(schedule)
    if len(schedule) != 168:
        raise RuntimeError(f'R8 epoch {epoch} has {len(schedule)} samples.')
    return schedule


def r8_schedule_audit():
    schedules = []
    repeated = []
    for epoch in range(1, R8_SCHEDULE_EPOCHS + 1):
        first = r8_build_epoch_schedule(epoch)
        second = r8_build_epoch_schedule(epoch)
        if first != second:
            raise RuntimeError(f'R8 epoch {epoch} schedule is not deterministic.')
        schedules.extend(first)
        repeated.extend(second)
    frame = pd.DataFrame(schedules)
    positive = frame[frame['sample_type'].astype(str) == 'positive'].copy()
    positive['component_key'] = (
        positive['subject_id'].astype(str) + ':'
        + positive['atomic_component_id'].astype(int).astype(str)
    )
    expected_keys = set(
        component_df[
            component_df['component_tier'].astype(str).isin(R8_TIER_QUOTAS)
        ].assign(component_key=lambda value:
            value['subject_id'].astype(str) + ':'
            + value['atomic_component_id'].astype(int).astype(str)
        )['component_key']
    )
    observed_keys = set(positive['component_key'])
    rows = []
    for tier, quota in R8_TIER_QUOTAS.items():
        tier_positive = positive[
            positive['component_tier'].astype(str) == tier
        ]
        counts = tier_positive.groupby('component_key').size()
        records = R8_COMPONENT_RECORDS_BY_TIER[tier]
        completed_cycles_are_unique = True
        for cycle_index, cycle in tier_positive.groupby('r8_tier_cycle_index'):
            keys = cycle['component_key'].tolist()
            if len(cycle) == len(records) and len(keys) != len(set(keys)):
                completed_cycles_are_unique = False
        rows.append({
            'component_tier': tier,
            'tier_components': len(records),
            'quota_per_epoch': int(quota),
            'total_selections_40_epochs': int(len(tier_positive)),
            'observed_components': int(counts.size),
            'never_selected_components': int(len(records) - counts.size),
            'minimum_selections': int(counts.min()),
            'median_selections': float(counts.median()),
            'mean_selections': float(counts.mean()),
            'maximum_selections': int(counts.max()),
            'selection_gini': r7d_gini(counts.to_numpy()),
            'maximum_minus_minimum_selections': int(
                counts.max() - counts.min()
            ),
            'complete_cycles_have_no_duplicates': bool(
                completed_cycles_are_unique
            ),
        })
    summary = pd.DataFrame(rows)
    per_epoch = positive.groupby('epoch').agg(
        positive_patches=('component_key', 'size'),
        unique_components=('component_key', 'nunique'),
        unique_subjects=('subject_id', 'nunique'),
    ).reset_index()
    summary.attrs['all_eligible_components_covered'] = (
        observed_keys == expected_keys
    )
    summary.attrs['minimum_unique_positive_subjects_per_epoch'] = int(
        per_epoch['unique_subjects'].min()
    )
    atomic_write_csv(frame, R8_SCHEDULE_PATH)
    atomic_write_csv(summary, R8_SCHEDULE_SUMMARY_PATH)
    return frame, summary, per_epoch, expected_keys, observed_keys


def r8_native_component_targets(row, local_origin, component_id):
    subject_id = str(row['subject_id'])
    binary, soft, support, spacing = r7d_component_support(row)
    labels, count = ndimage.label(
        support, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    expected = len(component_records.get(subject_id, []))
    if int(count) != int(expected):
        raise RuntimeError(
            f'{subject_id}: R8 reconstructed {count} components; expected {expected}.'
        )
    component_id = int(component_id)
    if not 1 <= component_id <= count:
        raise RuntimeError(f'{subject_id}: invalid component {component_id}.')
    selected = labels == component_id
    selected_canvas, start_a = symmetric_pad(
        selected.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    support_canvas, start_b = symmetric_pad(
        support.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(support, dtype=np.uint8),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if not (start_a == start_b == start_c):
        raise RuntimeError(f'{subject_id}: R8 target padding disagreement.')
    exact = crop_at(
        selected_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    full = crop_at(
        support_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    domain = crop_at(
        domain_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    if not bool(exact.any()):
        raise RuntimeError(f'{subject_id}: R8 native exact component is empty.')
    distance = ndimage.distance_transform_edt(~exact, sampling=spacing)
    negative = (
        (distance >= R8_NATIVE_INNER_RADIUS_MM)
        & (distance <= R8_NATIVE_OUTER_RADIUS_MM)
        & domain & ~full
    )
    if not bool(negative.any()):
        raise RuntimeError(f'{subject_id}: R8 native annulus is empty.')
    retained_fraction = float(exact.sum() / max(selected.sum(), 1))
    return {
        'native_component_target': exact[None],
        'native_component_negative_annulus': negative[None],
        'spacing': spacing,
        'component_native_voxels': int(selected.sum()),
        'component_patch_voxels': int(exact.sum()),
        'component_retained_fraction': retained_fraction,
        'negative_annulus_voxels': int(negative.sum()),
        'binary': binary,
        'soft': soft,
    }


def r8_native_target_probe(schedule_frame):
    probes = []
    positive = schedule_frame[
        schedule_frame['sample_type'].astype(str) == 'positive'
    ]
    for tier in R8_TIER_QUOTAS:
        candidates = positive[
            positive['component_tier'].astype(str) == tier
        ].head(8)
        for item in candidates.to_dict('records'):
            subject_id = str(item['subject_id'])
            row = fit_row_by_subject[subject_id]
            local_origin = pilot_origin_from_center(
                item['center_native'], pilot_mask_records[subject_id]['shape']
            )
            target = r8_native_component_targets(
                row, local_origin, item['atomic_component_id']
            )
            probes.append({
                'subject_id': subject_id,
                'component_tier': tier,
                'atomic_component_id': int(item['atomic_component_id']),
                'component_native_voxels': target['component_native_voxels'],
                'component_patch_voxels': target['component_patch_voxels'],
                'component_retained_fraction': target[
                    'component_retained_fraction'
                ],
                'negative_annulus_voxels': target[
                    'negative_annulus_voxels'
                ],
                'exact_and_annulus_disjoint': not bool(np.any(
                    target['native_component_target']
                    & target['native_component_negative_annulus']
                )),
                'target_arrays_read_split': 'fold0_fit',
                'locked_validation_target_read': False,
                'test_target_read': False,
            })
    frame = pd.DataFrame(probes)
    atomic_write_csv(frame, R8_NATIVE_TARGET_AUDIT_PATH)
    return frame


class NativeSatelliteProposalHead(nn.Module):
    def __init__(self, local_channels=32, hidden=R8_SATELLITE_FEATURE_CHANNELS):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(local_channels + 2, hidden, 1, bias=False),
            nn.GroupNorm(group_count(hidden), hidden), nn.GELU(),
        )
        self.refine = ResidualBlock3D(hidden, hidden, dropout=0.05)
        self.output = nn.Conv3d(hidden, 1, 1)

    def forward(self, local_feature, coarse_proposal, arterial_local):
        feature = self.project(torch.cat([
            local_feature, torch.sigmoid(coarse_proposal), arterial_local,
        ], dim=1))
        return self.output(self.refine(feature))


class VascuFateNetR8(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.base = VascuFateNetV22(schema)
        self.satellite_head = NativeSatelliteProposalHead()
        self.satellite_gate = ConfidenceGate(
            confidence_dim=5, output_dim=1,
            bound=R8_SATELLITE_GATE_BOUND, hidden_dim=32,
        )
        self._decoder_feature = None
        self._decoder_hook = self.base.local_model.decoder1.register_forward_hook(
            self._capture_decoder_feature
        )
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def _capture_decoder_feature(self, module, inputs, output):
        self._decoder_feature = output

    def load_r5_base_state(self, state):
        self.base.load_state_dict(state, strict=True)

    def activate_satellite_gate_for_gradient_test(self, fraction=0.5):
        self.satellite_gate.activate_for_gradient_test(fraction)

    def reset_satellite_gate_to_zero(self):
        with torch.no_grad():
            self.satellite_gate.network[-1].weight.zero_()
            self.satellite_gate.network[-1].bias.zero_()

    def forward(self, **kwargs):
        self._decoder_feature = None
        output = self.base(**kwargs)
        if self._decoder_feature is None:
            raise RuntimeError('R8 decoder feature hook did not fire.')
        arterial_local = sample_global_to_local(
            output['arterial_global_map'][:, None],
            kwargs['local_origin_in_canvas'],
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        if bool(kwargs.get('force_global_dropout', False)):
            coarse = torch.zeros_like(output['global_proposal_local'])
            arterial_local = torch.zeros_like(arterial_local)
        else:
            coarse = output['global_proposal_local']
        satellite_logits = self.satellite_head(
            self._decoder_feature, coarse, arterial_local
        )
        confidence = torch.cat([
            kwargs['confidence_features'],
            torch.sigmoid(output['lesion_presence_logit'])[:, None],
        ], dim=-1)
        gate = self.satellite_gate(confidence)
        base_logits = output['logits']
        fused_logits = base_logits + gate[..., None, None, None] * satellite_logits
        self._decoder_feature = None
        return {
            **output,
            'base_logits': base_logits,
            'satellite_logits': satellite_logits,
            'satellite_gate': gate,
            'logits': fused_logits,
        }


def r8_satellite_component_loss(
    logits, exact_target, negative_annulus, component_valid,
):
    logits = logits.float()
    exact_target = exact_target.bool()
    negative_annulus = negative_annulus.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    rows = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]]
        negative = logits[batch_index][negative_annulus[batch_index]]
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError('R8 component loss received an empty region.')
        hard_count = min(
            int(negative.numel()),
            max(32, 4 * int(positive.numel())),
        )
        hard_negative = torch.topk(negative, hard_count).values
        positive_bce = F.softplus(-positive).mean()
        negative_bce = F.softplus(hard_negative).mean()
        rank = F.relu(
            1.0 - positive.mean() + hard_negative.mean()
        )
        absolute_floor = F.softplus(
            R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR - positive.mean()
        )
        loss = (
            0.35 * positive_bce + 0.30 * negative_bce
            + 0.20 * rank + 0.15 * absolute_floor
        )
        losses.append(loss)
        rows.append({
            'positive_voxels': int(positive.numel()),
            'negative_voxels': int(negative.numel()),
            'hard_negative_voxels': int(hard_count),
            'positive_logit_mean': float(positive.mean().detach().cpu()),
            'hard_negative_logit_mean': float(
                hard_negative.mean().detach().cpu()
            ),
            'relative_logit_margin': float(
                (positive.mean() - hard_negative.mean()).detach().cpu()
            ),
            'positive_bce': float(positive_bce.detach().cpu()),
            'hard_negative_bce': float(negative_bce.detach().cpu()),
            'rank_loss': float(rank.detach().cpu()),
            'absolute_positive_floor_loss': float(
                absolute_floor.detach().cpu()
            ),
            'total_satellite_loss': float(loss.detach().cpu()),
        })
    if not losses:
        return logits.sum() * 0.0, []
    return torch.stack(losses).mean(), rows


def r8_model_input_batch(sample, device):
    return {
        key: torch.from_numpy(np.asarray(sample[key])[None]).to(device)
        for key in MODEL_INPUT_KEYS_V22
    }


def r8_gradient_norm(module):
    values = [
        parameter.grad.detach().float().norm().cpu().item()
        for parameter in module.parameters()
        if parameter.grad is not None
    ]
    if not values:
        return 0.0, False, True
    finite = bool(np.isfinite(values).all())
    return float(np.linalg.norm(values)), True, finite


def run_r8_dual_resolution_preflight():
    if r8_existing_preflight_complete():
        gate = load_json(R8_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Total parameters: {gate["total_parameters"]:,}')
        print(f'Preflight gate:   {R8_GATE_PATH}')
        print('No source checkpoint or preflight artifact was rewritten.')
        return

    r7_failure_gate = load_json(R7_FAILURE_GATE_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r5_recomputed = dict(
        r7_result['r5_recomputed_under_r7_metric_contract']
    )
    r5_mean_dice = float(r5_recomputed['positive_mean_dice'])
    r5_global_dice = float(r5_recomputed['positive_global_dice'])
    r8_mean_dice_eligibility_floor = r5_mean_dice - 0.005
    r8_global_dice_eligibility_floor = r5_global_dice - 0.010
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_failure_sha = file_sha256(R7_FAILURE_GATE_PATH)
    upstream_checks = {
        'r7_failure_diagnostic_complete': (
            r7_failure_gate.get('diagnostic_complete') is True
        ),
        'r7_failure_diagnostic_passed': (
            r7_failure_gate.get('diagnostic_passed') is True
        ),
        'r8_preflight_allowed_by_diagnostic': (
            r7_failure_gate.get('r8_preflight_implementation_allowed') is True
        ),
        'r8_training_was_locked_by_diagnostic': (
            r7_failure_gate.get('r8_training_authorized') is False
        ),
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r5_recomputed_baseline_is_finite': bool(np.isfinite([
            r5_mean_dice, r5_global_dice,
        ]).all()),
        'r5_checkpoint_hash_matches': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'fold0_fit_count_is_82': len(fold0_fit_df) == 82,
        'fold0_heldout_targets_not_used': len(fold0_heldout_df) == 20,
        'locked_validation_targets_not_used': len(locked_validation_df) == 23,
        'sealed_test_targets_not_used': len(sealed_test_df) == 23,
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError('R8 upstream checks failed: ' + ', '.join(failed))

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT STARTING')
    print('=' * 80)
    print('Warm-start source:                  completed r5 best checkpoint')
    print('R7 checkpoint used for weights:     False')
    print('Target access:                      fold-0 fit probes only')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer / training steps:         0 / 0')

    schedule, schedule_summary, per_epoch, expected_keys, observed_keys = (
        r8_schedule_audit()
    )
    target_probe = r8_native_target_probe(schedule)

    positive_schedule = schedule[
        schedule['sample_type'].astype(str) == 'positive'
    ]
    smoke_candidates = positive_schedule[
        positive_schedule['component_tier'].astype(str).isin(['micro', 'tiny'])
    ]
    smoke_entry = None
    for item in smoke_candidates.to_dict('records'):
        if as_bool(fit_row_by_subject[str(item['subject_id'])][
            'trial8_atlas_enabled'
        ]):
            smoke_entry = item
            break
    if smoke_entry is None:
        raise RuntimeError('R8 found no atlas-enabled micro/tiny smoke case.')
    subject_id = str(smoke_entry['subject_id'])
    row = fit_row_by_subject[subject_id]
    local_origin = pilot_origin_from_center(
        smoke_entry['center_native'], pilot_mask_records[subject_id]['shape']
    )
    dataset = Trial8V22Fold0Dataset()
    dataset.epoch = int(smoke_entry['epoch'])
    dataset.schedule = [smoke_entry]
    sample = dataset[0]
    native_target = r8_native_component_targets(
        row, local_origin, smoke_entry['atomic_component_id']
    )

    if not torch.cuda.is_available():
        raise RuntimeError('R8 preflight requires an L4-class CUDA runtime.')
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < 20.0:
        raise RuntimeError(
            f'R8 preflight expected >=20 GB GPU, observed {gpu_capacity_gb:.2f} GB.'
        )
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'r5_checkpoint_pilot_version_matches': (
            checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'r5_checkpoint_code_revision_matches': (
            checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
        ),
        'r5_checkpoint_parameter_count_matches': int(
            checkpoint.get('parameter_count', -1)
        ) == EXPECTED_V22_PARAMETERS,
        'r5_checkpoint_contains_ema_state': 'ema_model_state' in checkpoint,
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 warm start: ' + ', '.join(failed))
    batch = r8_model_input_batch(sample, device)
    force_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }

    reference = VascuFateNetV22(pilot_schema).to(device).eval()
    reference.load_state_dict(checkpoint['ema_model_state'], strict=True)
    for parameter in reference.parameters():
        parameter.requires_grad_(False)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        reference_output = reference(**batch, **force_flags)
    reference_cpu = {
        key: value.detach().float().cpu()
        for key, value in reference_output.items()
        if torch.is_tensor(value)
    }
    del reference, reference_output
    gc.collect()
    torch.cuda.empty_cache()

    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(checkpoint['ema_model_state'])
    del checkpoint
    model.eval()
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    base_parameters = sum(parameter.numel() for parameter in model.base.parameters())
    satellite_head_parameters = sum(
        parameter.numel() for parameter in model.satellite_head.parameters()
    )
    satellite_gate_parameters = sum(
        parameter.numel() for parameter in model.satellite_gate.parameters()
    )
    new_parameters = satellite_head_parameters + satellite_gate_parameters
    parameter_frame = pd.DataFrame([
        {'component': 'preserved_r5_v2_2_base', 'parameters': base_parameters},
        {'component': 'native_satellite_head', 'parameters': satellite_head_parameters},
        {'component': 'bounded_satellite_gate', 'parameters': satellite_gate_parameters},
        {'component': 'new_r8_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
    ])
    atomic_write_csv(parameter_frame, R8_PARAMETER_PATH)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        neutral_output = model(**batch, **force_flags)
    peak_eval_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    preserved_keys = [
        'logits', 'auxiliary_half_logits', 'auxiliary_quarter_logits',
        'territory_node_logits', 'global_graph_context',
        'global_proposal_logits', 'lesion_presence_logit',
        'log_volume_prediction',
    ]
    neutrality_rows = []
    for key in preserved_keys:
        difference = float(torch.max(torch.abs(
            neutral_output[key].detach().float().cpu() - reference_cpu[key]
        )))
        neutrality_rows.append({
            'output': key,
            'maximum_absolute_difference_from_r5': difference,
            'exactly_preserved_with_zero_gate': difference <= 1e-6,
        })
    neutrality_frame = pd.DataFrame(neutrality_rows)
    atomic_write_csv(neutrality_frame, R8_NEUTRALITY_PATH)

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'base_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_gate': (1, 1),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
    }
    output_rows = []
    for key, expected in expected_shapes.items():
        observed = tuple(neutral_output[key].shape)
        output_rows.append({
            'output': key,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(neutral_output[key]).all().item()),
        })
    output_frame = pd.DataFrame(output_rows)
    atomic_write_csv(output_frame, R8_OUTPUT_PATH)

    zero_gate = float(neutral_output['satellite_gate'].abs().max().cpu())
    satellite_std = float(
        neutral_output['satellite_logits'].float().std().cpu()
    )
    model.activate_satellite_gate_for_gradient_test(0.5)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        activated_output = model(**batch, **force_flags)
    activated_difference = float(torch.max(torch.abs(
        activated_output['logits'].float()
        - activated_output['base_logits'].float()
    )).cpu())
    model.reset_satellite_gate_to_zero()
    del activated_output

    model.train()
    model.zero_grad(set_to_none=True)
    exact_target = torch.from_numpy(
        native_target['native_component_target'][None]
    ).to(device)
    negative_target = torch.from_numpy(
        native_target['native_component_negative_annulus'][None]
    ).to(device)
    component_valid = torch.ones(1, dtype=torch.bool, device=device)
    local_binary = torch.from_numpy(
        np.asarray(sample['target_binary'])[None]
    ).to(device)
    global_target = torch.from_numpy(
        np.asarray(sample['global_target'])[None]
    ).to(device)
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        gradient_output = model(**batch, **force_flags)
        main_loss = (
            0.60 * pilot_soft_dice_loss(
                gradient_output['logits'], local_binary
            ).mean()
            + 0.40 * pilot_focal_bce_loss(
                gradient_output['logits'], local_binary
            ).mean()
        )
        coarse_loss = (
            0.70 * pilot_soft_dice_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
            + 0.30 * pilot_focal_bce_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
        )
        satellite_loss, satellite_loss_rows = r8_satellite_component_loss(
            gradient_output['satellite_logits'], exact_target,
            negative_target, component_valid,
        )
        smoke_loss = main_loss + 0.20 * coarse_loss + 0.10 * satellite_loss
    smoke_loss.backward()
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_groups = {
        'preserved_local_segmentation_head': (
            model.base.local_model.segmentation_head
        ),
        'preserved_global_proposal_stream': model.base.global_stream,
        'native_satellite_head': model.satellite_head,
        'bounded_satellite_gate': model.satellite_gate,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_groups.items():
        norm, observed, finite = r8_gradient_norm(module)
        gradient_rows.append({
            'component': name,
            'gradient_norm': norm,
            'gradient_observed': observed,
            'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_frame = pd.DataFrame(gradient_rows)
    atomic_write_csv(gradient_frame, R8_GRADIENT_PATH)
    loss_frame = pd.DataFrame([{
        'subject_id': subject_id,
        'component_tier': str(smoke_entry['component_tier']),
        'component_id': int(smoke_entry['atomic_component_id']),
        'main_loss': float(main_loss.detach().float().cpu()),
        'coarse_whole_mask_loss': float(coarse_loss.detach().float().cpu()),
        'native_satellite_loss': float(
            satellite_loss.detach().float().cpu()
        ),
        'total_smoke_loss': float(smoke_loss.detach().float().cpu()),
        **satellite_loss_rows[0],
    }])
    atomic_write_csv(loss_frame, R8_LOSS_PATH)
    model.zero_grad(set_to_none=True)

    architecture = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'warm_start': {
            'source': 'completed_r5_best_ema',
            'checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'checkpoint_sha256': r5_sha_before,
            'r7_weights_used': False,
            'zero_gate_output_equivalence_required': True,
        },
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'native_satellite_head': {
            'input': 'native-resolution_r5_decoder_feature_plus_coarse_proposal_plus_arterial_map',
            'input_spatial_shape': list(LOCAL_PATCH_SIZE),
            'hidden_channels': R8_SATELLITE_FEATURE_CHANNELS,
            'output': 'native_resolution_satellite_logits',
            'direct_graph_to_logit_residual': False,
        },
        'fusion': {
            'type': 'zero_initialized_bounded_scalar_residual',
            'bound': R8_SATELLITE_GATE_BOUND,
            'gate_context': 'registration_confidence_plus_lesion_presence',
            'initial_final_output': 'exact_r5_logits',
        },
        'coarse_global_role': (
            'whole_brain_burden_and_broad_localization_only; '
            'no_r7_coarse_component_contrastive_loss'
        ),
        'clinical_and_arterial_role': (
            'preserved_bounded_v2_2_context; no treatment/outcome leakage'
        ),
    }
    training_policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'experiment': 'R8_fold0_pilot_only_after_separate_training_preflight',
        'component_cycle': {
            'positive_patches_per_epoch': R8_COMPONENT_POSITIVE_PATCHES,
            'tier_quotas': R8_TIER_QUOTAS,
            'rule': 'within_each_tier_every_component_once_before_repeat',
            'schedule_epochs_audited': R8_SCHEDULE_EPOCHS,
        },
        'optimization_stages': [
            {
                'epochs': '1-4',
                'trainable': ['native_satellite_head'],
                'frozen': ['r5_base', 'satellite_gate'],
                'purpose': 'learn_absolute_native_component_evidence_before_fusion',
            },
            {
                'epochs': '5-12',
                'trainable': ['native_satellite_head', 'satellite_gate'],
                'frozen': ['r5_base'],
                'purpose': 'learn_bounded_residual_without_erasing_r5_behavior',
            },
            {
                'epochs': '13+',
                'trainable': ['all'],
                'learning_rate_policy': (
                    'r5_base_2e-5_satellite_2e-4_gate_1e-4'
                ),
                'purpose': 'low_rate_joint_calibration',
            },
        ],
        'loss_contract': {
            'main_segmentation': 'preserved_r5_dice_plus_focal',
            'coarse_proposal': 'whole_mask_only',
            'native_satellite': (
                'per_component_positive_bce_plus_hard_negative_bce_plus_'
                'ranking_plus_absolute_positive_logit_floor'
            ),
            'coarse_component_contrastive': False,
            'volume_case_weights': False,
            'topk_ohem': False,
        },
        'checkpoint_eligibility': {
            'r5_recomputed_mean_dice': r5_mean_dice,
            'r5_recomputed_global_dice': r5_global_dice,
            'mean_dice_minimum': r8_mean_dice_eligibility_floor,
            'global_dice_minimum': r8_global_dice_eligibility_floor,
            'small_component_improvement_required': True,
            'deployment_metrics_only': True,
            'oracle_volume_budget_used_for_selection': False,
            'selection_order': [
                'eligibility_guards', 'mean_dice', 'small_case_dice',
                'small_component_recall', 'lesion_f1',
            ],
        },
        'fold0_heldout_targets_for_training': False,
        'locked_validation_targets_for_training': False,
        'test_targets_for_training': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(architecture, R8_ARCHITECTURE_PATH)
    atomic_write_json(training_policy, R8_TRAINING_POLICY_PATH)

    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    preflight_checks = {
        **upstream_checks,
        **checkpoint_checks,
        'tier_quotas_sum_to_112': sum(R8_TIER_QUOTAS.values()) == 112,
        'schedule_has_40_epochs': schedule['epoch'].nunique() == 40,
        'every_epoch_has_168_samples': bool(
            (schedule.groupby('epoch').size() == 168).all()
        ),
        'every_epoch_has_112_positive_samples': bool(
            (schedule[schedule['sample_type'] == 'positive']
             .groupby('epoch').size() == 112).all()
        ),
        'all_eligible_components_are_covered': observed_keys == expected_keys,
        'no_eligible_component_is_never_selected': bool(
            (schedule_summary['never_selected_components'] == 0).all()
        ),
        'component_selection_counts_differ_by_at_most_one': bool(
            (schedule_summary['maximum_minus_minimum_selections'] <= 1).all()
        ),
        'complete_tier_cycles_have_no_duplicates': bool(
            schedule_summary['complete_cycles_have_no_duplicates'].all()
        ),
        'component_cycle_gini_below_0p10': bool(
            (schedule_summary['selection_gini'] < 0.10).all()
        ),
        'native_target_probes_cover_four_tiers': (
            target_probe['component_tier'].nunique() == 4
        ),
        'all_native_target_probes_are_nonempty': bool(
            (target_probe['component_patch_voxels'] > 0).all()
        ),
        'all_native_annuli_are_nonempty': bool(
            (target_probe['negative_annulus_voxels'] > 0).all()
        ),
        'all_native_targets_and_annuli_are_disjoint': bool(
            target_probe['exact_and_annulus_disjoint'].all()
        ),
        'all_native_target_probes_retain_at_least_95_percent': bool(
            (target_probe['component_retained_fraction'] >= 0.95).all()
        ),
        'preserved_base_parameter_count_matches_r5': (
            base_parameters == EXPECTED_V22_PARAMETERS
        ),
        'new_parameter_count_within_limit': (
            0 < new_parameters <= R8_MAXIMUM_NEW_PARAMETERS
        ),
        'all_zero_gate_outputs_match_r5': bool(
            neutrality_frame['exactly_preserved_with_zero_gate'].all()
        ),
        'satellite_gate_is_exactly_zero_initially': zero_gate == 0.0,
        'satellite_logits_are_nonconstant': satellite_std > 1e-6,
        'activated_satellite_gate_changes_final_logits': (
            activated_difference > 1e-7
        ),
        'all_output_shapes_match': bool(output_frame['shape_matches'].all()),
        'all_outputs_are_finite': bool(output_frame['finite'].all()),
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss).item()),
        'satellite_loss_contains_absolute_positive_floor': (
            float(loss_frame['absolute_positive_floor_loss'].iloc[0]) > 0.0
        ),
        **gradient_checks,
        'peak_eval_memory_below_20gb': peak_eval_memory_gb < 20.0,
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'r7_checkpoint_not_loaded_for_weights': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in preflight_checks.items()
    ])
    atomic_write_csv(checks_frame, R8_CHECKS_PATH)
    preflight_passed = all(preflight_checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R8_OUTPUT_ARTIFACTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_used_for_weights': False,
        'r7_failure_diagnostic_path': str(R7_FAILURE_GATE_PATH),
        'r7_failure_diagnostic_sha256': r7_failure_sha,
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'gpu': gpu_name,
        'gpu_capacity_gb': gpu_capacity_gb,
        'peak_eval_memory_gb': peak_eval_memory_gb,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'schedule_components': len(expected_keys),
        'minimum_unique_positive_subjects_per_epoch': int(
            per_epoch['unique_subjects'].min()
        ),
        'native_target_probes': len(target_probe),
        'optimizer_steps': 0,
        'training_performed': False,
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'r8_pilot_implementation_allowed': bool(preflight_passed),
        'r8_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'checks': preflight_checks,
        'artifact_sha256': artifact_hashes,
        'architecture_path': str(R8_ARCHITECTURE_PATH),
        'training_policy_path': str(R8_TRAINING_POLICY_PATH),
    }
    atomic_write_json(gate, R8_GATE_PATH)
    del model, neutral_output, gradient_output, batch
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in preflight_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR8 component-cycle summary:')
    display(schedule_summary)
    print('\nR8 parameter count:')
    display(parameter_frame)
    print('\nR8 output shapes:')
    display(output_frame)
    print('\nR8 gradient smoke:')
    display(gradient_frame)
    print('\nR8 preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Warm-start checkpoint:              r5 epoch {r5_result["best_epoch"]}')
    print('R7 weights used:                    False')
    print(f'Preserved base parameters:          {base_parameters:,}')
    print(f'New satellite parameters:           {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Eligible components cycled:         {len(expected_keys)}')
    print(f'Minimum unique subjects per epoch:  {int(per_epoch["unique_subjects"].min())}')
    print(f'Peak gradient memory GB:            {peak_gradient_memory_gb:.3f}')
    print('Coarse component contrastive loss:  disabled')
    print('Native absolute calibration loss:   enabled')
    print('Initial final logits equal r5:      True')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Training performed:                 False')
    print('R8 pilot implementation allowed:    True')
    print('R8 pilot training authorized:       False')
    print('Full five-fold training authorized: False')
    print(f'Architecture:                       {R8_ARCHITECTURE_PATH}')
    print(f'Training policy:                    {R8_TRAINING_POLICY_PATH}')
    print(f'Preflight gate:                     {R8_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 R8 pilot loader/loss/training '
        'preflight from this frozen contract; do not train yet.'
    )


run_r8_dual_resolution_preflight()












R8P_PREFLIGHT_VERSION = 'v1_stagewise_native_satellite_pilot_contract'
R8P_CODE_REVISION = 'r1_negative_hard_background_pareto_resume_guard'
R8P_PILOT_VERSION = 'v2_3_fold0_dual_resolution_satellite_r8'
R8P_PILOT_CODE_REVISION = 'r1_r5_warmstart_stagewise_native_satellite'
R8P_SEED = 20261008
R8P_MAX_EPOCHS = 40
R8P_MINIMUM_EPOCHS = 20
R8P_EARLY_STOPPING_PATIENCE = 10
R8P_EARLY_STOPPING_MINIMUM_DELTA = 0.002
R8P_ACCUMULATION_STEPS = 4
R8P_HEAD_LR = 2.0e-4
R8P_GATE_LR = 1.0e-4
R8P_BASE_LR = 2.0e-5
R8P_MINIMUM_LR_FRACTION = 0.10
R8P_WEIGHT_DECAY = 1.0e-4
R8P_GRADIENT_CLIP_NORM = 2.0
R8P_SATELLITE_LOSS_COEFFICIENT = 0.10
R8P_NEGATIVE_BACKGROUND_WEIGHT = 0.25
R8P_NEGATIVE_BACKGROUND_TOPK = 4096
R8P_VALIDATION_OVERLAP = 0.50
R8P_LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
R8P_PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
R8P_EXPECTED_PARAMETERS = 16_885_385

R8P_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / R8_MODEL_VERSION / R8P_PREFLIGHT_VERSION
)
R8P_ROOT.mkdir(parents=True, exist_ok=True)
R8P_POLICY_PATH = R8P_ROOT / 'r8_fold0_pilot_policy.json'
R8P_RESUME_PATH = R8P_ROOT / 'r8_resume_contract.json'
R8P_LOADER_AUDIT_PATH = R8P_ROOT / 'r8_loader_case_audit.csv'
R8P_BATCH_AUDIT_PATH = R8P_ROOT / 'r8_batch_tensor_audit.csv'
R8P_STAGE_AUDIT_PATH = R8P_ROOT / 'r8_stage_trainability_audit.csv'
R8P_LR_AUDIT_PATH = R8P_ROOT / 'r8_learning_rate_audit.csv'
R8P_OPTIMIZER_AUDIT_PATH = R8P_ROOT / 'r8_optimizer_group_audit.csv'
R8P_LOSS_AUDIT_PATH = R8P_ROOT / 'r8_loss_smoke_audit.csv'
R8P_GRADIENT_AUDIT_PATH = R8P_ROOT / 'r8_stage_gradient_audit.csv'
R8P_CACHE_AUDIT_PATH = R8P_ROOT / 'r8_cached_inference_audit.csv'
R8P_SELECTION_AUDIT_PATH = R8P_ROOT / 'r8_checkpoint_policy_audit.csv'
R8P_CHECKS_PATH = R8P_ROOT / 'r8_fold0_pilot_preflight_checks.csv'
R8P_GATE_PATH = R8P_ROOT / 'r8_fold0_pilot_preflight_gate.json'

R8P_OUTPUT_ARTIFACTS = [
    R8P_POLICY_PATH, R8P_RESUME_PATH, R8P_LOADER_AUDIT_PATH,
    R8P_BATCH_AUDIT_PATH, R8P_STAGE_AUDIT_PATH, R8P_LR_AUDIT_PATH,
    R8P_OPTIMIZER_AUDIT_PATH, R8P_LOSS_AUDIT_PATH,
    R8P_GRADIENT_AUDIT_PATH, R8P_CACHE_AUDIT_PATH,
    R8P_SELECTION_AUDIT_PATH, R8P_CHECKS_PATH,
]

R8P_PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / R8_MODEL_VERSION / R8P_PILOT_VERSION
)
R8P_PILOT_CONFIG_PATH = R8P_PILOT_ROOT / 'pilot_config.json'
R8P_PILOT_START_GATE_PATH = R8P_PILOT_ROOT / 'pilot_start_gate.json'
R8P_LAST_CHECKPOINT_PATH = R8P_PILOT_ROOT / 'last_checkpoint.pt'
R8P_BEST_CHECKPOINT_PATH = R8P_PILOT_ROOT / 'best_checkpoint.pt'
R8P_HISTORY_PATH = R8P_PILOT_ROOT / 'training_history.csv'
R8P_RESULT_GATE_PATH = R8P_PILOT_ROOT / 'pilot_result_gate.json'


def r8p_existing_preflight_complete():
    if not R8P_GATE_PATH.exists():
        return False
    gate = load_json(R8P_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('preflight_version') == R8P_PREFLIGHT_VERSION
        and gate.get('code_revision') == R8P_CODE_REVISION
        and gate.get('r8_architecture_gate_sha256') == file_sha256(R8_GATE_PATH)
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R8P_OUTPUT_ARTIFACTS
    )


def r8p_parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in sorted(model.named_parameters()):
        digest.update(name.encode('utf-8'))
        digest.update(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        )
    return digest.hexdigest()


def r8p_stage(epoch):
    epoch = int(epoch)
    if not 1 <= epoch <= R8P_MAX_EPOCHS:
        raise ValueError(f'R8 pilot epoch {epoch} is outside 1..40.')
    if epoch <= 4:
        return 'satellite_head_only'
    if epoch <= 12:
        return 'satellite_head_and_gate'
    return 'low_rate_joint_calibration'


def r8p_branch_lr(epoch, start_epoch, peak_lr):
    epoch = int(epoch)
    start_epoch = int(start_epoch)
    if epoch < start_epoch:
        return 0.0
    if epoch <= start_epoch + 1:
        return float(peak_lr * (epoch - start_epoch + 1) / 2.0)
    minimum = peak_lr * R8P_MINIMUM_LR_FRACTION
    denominator = max(R8P_MAX_EPOCHS - (start_epoch + 1), 1)
    progress = np.clip(
        (epoch - (start_epoch + 1)) / denominator, 0.0, 1.0
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * float(progress)))
    return float(minimum + (peak_lr - minimum) * cosine)


def r8p_learning_rates(epoch):
    return {
        'r5_base': r8p_branch_lr(epoch, 13, R8P_BASE_LR),
        'native_satellite_head': r8p_branch_lr(epoch, 1, R8P_HEAD_LR),
        'bounded_satellite_gate': r8p_branch_lr(epoch, 5, R8P_GATE_LR),
    }


def r8p_set_requires_grad(module, value):
    for parameter in module.parameters():
        parameter.requires_grad_(bool(value))


def r8p_apply_stage(model, epoch, optimizer=None):
    stage = r8p_stage(epoch)
    model.train()
    base_trainable = stage == 'low_rate_joint_calibration'
    gate_trainable = stage != 'satellite_head_only'
    r8p_set_requires_grad(model.base, base_trainable)
    r8p_set_requires_grad(model.satellite_head, True)
    r8p_set_requires_grad(model.satellite_gate, gate_trainable)
    if not base_trainable:
        model.base.eval()
    if not gate_trainable:
        model.satellite_gate.eval()
    learning_rates = r8p_learning_rates(epoch)
    if optimizer is not None:
        observed_names = set()
        for group in optimizer.param_groups:
            name = str(group['group_name'])
            observed_names.add(name)
            group['lr'] = float(learning_rates[name])
        if observed_names != set(learning_rates):
            raise RuntimeError('R8 optimizer groups do not match LR policy.')
    return stage, learning_rates


def r8p_optimizer_groups(model):
    groups = [
        {
            'group_name': 'r5_base',
            'params': list(model.base.parameters()),
            'lr': 0.0,
            'weight_decay': R8P_WEIGHT_DECAY,
        },
        {
            'group_name': 'native_satellite_head',
            'params': list(model.satellite_head.parameters()),
            'lr': R8P_HEAD_LR / 2.0,
            'weight_decay': R8P_WEIGHT_DECAY,
        },
        {
            'group_name': 'bounded_satellite_gate',
            'params': list(model.satellite_gate.parameters()),
            'lr': 0.0,
            'weight_decay': 0.0,
        },
    ]
    identifiers = [
        id(parameter) for group in groups for parameter in group['params']
    ]
    if len(identifiers) != len(set(identifiers)):
        raise RuntimeError('R8 optimizer parameter groups overlap.')
    model_identifiers = {id(parameter) for parameter in model.parameters()}
    if set(identifiers) != model_identifiers:
        raise RuntimeError('R8 optimizer groups do not cover the model exactly.')
    return groups


class Trial8R8Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = r8_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = dict(self.schedule[int(index)])
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        positive = str(entry['sample_type']) == 'positive'
        if positive:
            native = r8_native_component_targets(
                row, local_origin, entry['atomic_component_id']
            )
            if not bool((targets['target_soft'] > 0).any()):
                raise RuntimeError(
                    f'{subject_id}: R8 positive patch lost soft support.'
                )
        else:
            if bool((targets['target_soft'] > 0).any()):
                raise RuntimeError(
                    f'{subject_id}: R8 negative patch contains target.'
                )
            native = {
                'native_component_target': np.zeros(
                    (1, *LOCAL_PATCH_SIZE), dtype=bool
                ),
                'native_component_negative_annulus': np.zeros(
                    (1, *LOCAL_PATCH_SIZE), dtype=bool
                ),
                'component_patch_voxels': 0,
                'negative_annulus_voxels': 0,
                'component_retained_fraction': 1.0,
            }
        rng = np.random.default_rng(
            R8P_SEED + self.epoch * 10_000_019 + int(index) * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'native_component_target': native['native_component_target'],
            'native_component_negative_annulus': native[
                'native_component_negative_annulus'
            ],
            'native_component_valid': np.asarray(positive, dtype=bool),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
            'component_patch_voxels': np.asarray(
                native['component_patch_voxels'], dtype=np.int64
            ),
            'negative_annulus_voxels': np.asarray(
                native['negative_annulus_voxels'], dtype=np.int64
            ),
            'component_retained_fraction': np.asarray(
                native['component_retained_fraction'], dtype=np.float32
            ),
        })
        return sample


def r8p_negative_background_loss(logits, component_valid):
    valid = component_valid.bool().reshape(-1)
    losses = []
    selected_counts = []
    for batch_index in range(logits.shape[0]):
        if bool(valid[batch_index]):
            continue
        flattened = logits[batch_index].float().reshape(-1)
        hard_count = min(R8P_NEGATIVE_BACKGROUND_TOPK, flattened.numel())
        hard = torch.topk(flattened, k=hard_count, largest=True).values
        losses.append(F.softplus(hard).mean())
        selected_counts.append(hard_count)
    if not losses:
        return logits.float().sum() * 0.0, 0
    return torch.stack(losses).mean(), int(sum(selected_counts))


def r8p_whole_proposal_loss(logits, target):
    dice = pilot_soft_dice_loss(logits, target).mean()
    focal = pilot_focal_bce_loss(logits, target).mean()
    return 0.65 * dice + 0.35 * focal, dice, focal


def r8p_compute_training_loss(
    model, cpu_batch, device, epoch, dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    if dropout_flags_override is None:
        rng = np.random.default_rng(R8P_SEED + int(epoch) * 1_000_003)
        flags = pilot_dropout_flags(rng)
    else:
        flags = {name: bool(value) for name, value in dropout_flags_override.items()}
    required_flags = {
        'force_atlas_dropout', 'force_vascular_dropout',
        'force_occlusion_dropout', 'force_clinical_dropout',
        'force_global_dropout', 'force_arterial_dropout',
    }
    if set(flags) != required_flags:
        raise RuntimeError('R8 dropout flags do not match the frozen schema.')
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool()
        & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    proposal_loss, proposal_dice, proposal_focal = r8p_whole_proposal_loss(
        output['global_proposal_logits'], batch['global_target'].float()
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].float().sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], batch['global_target'] > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'],
        global_territory_target, arterial_valid,
    )
    satellite_positive, satellite_rows = r8_satellite_component_loss(
        output['satellite_logits'], batch['native_component_target'],
        batch['native_component_negative_annulus'],
        batch['native_component_valid'],
    )
    satellite_negative, hard_background_voxels = (
        r8p_negative_background_loss(
            output['satellite_logits'], batch['native_component_valid']
        )
    )
    satellite_objective = (
        satellite_positive
        + R8P_NEGATIVE_BACKGROUND_WEIGHT * satellite_negative
    )
    preserved_objective = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    total = (
        preserved_objective
        + R8P_SATELLITE_LOSS_COEFFICIENT * satellite_objective
    )
    parts = {
        'total_loss': total,
        'preserved_objective': preserved_objective,
        'main_loss': main_loss,
        **main_parts,
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'proposal_whole_mask_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_focal,
        'presence_loss': presence_loss,
        'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'satellite_positive_loss': satellite_positive,
        'satellite_negative_background_loss': satellite_negative,
        'satellite_objective': satellite_objective,
        'satellite_loss_coefficient': torch.as_tensor(
            R8P_SATELLITE_LOSS_COEFFICIENT,
            device=total.device, dtype=torch.float32,
        ),
        'hard_background_voxels': torch.as_tensor(
            float(hard_background_voxels),
            device=total.device, dtype=torch.float32,
        ),
        'native_components_in_batch': batch[
            'native_component_valid'
        ].float().sum(),
    }
    return total, parts, satellite_rows, output


def r8p_cached_local_from_global(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    model._decoder_feature = None
    base_output = pilot_local_from_cache(
        model.base, batch, cache,
        force_global_dropout=force_global_dropout,
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    if model._decoder_feature is None:
        raise RuntimeError('R8 cached path did not capture decoder feature.')
    coarse = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    if bool(force_global_dropout):
        coarse = torch.zeros_like(coarse)
        arterial_local = torch.zeros_like(arterial_local)
    satellite_logits = model.satellite_head(
        model._decoder_feature, coarse, arterial_local
    )
    confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    gate = model.satellite_gate(confidence)
    base_logits = base_output['logits']
    result = {
        **base_output,
        'base_logits': base_logits,
        'satellite_logits': satellite_logits,
        'satellite_gate': gate,
        'logits': base_logits + gate[..., None, None, None] * satellite_logits,
        'arterial_global_map': cache['arterial_map'],
        'global_proposal_local': coarse,
    }
    model._decoder_feature = None
    return result


def r8p_r5_small_component_recall(r5_baseline):
    rows = r5_baseline.get('component_tier_metrics', [])
    selected = [
        row for row in rows
        if str(row.get('component_tier')) in set(COMPONENT_TIERS[:4])
    ]
    denominator = sum(int(row.get('components', 0)) for row in selected)
    if denominator <= 0:
        raise RuntimeError('R5 baseline has no small-component audit rows.')
    numerator = sum(
        int(row.get('components', 0))
        * float(row.get('final_component_recall', 0.0))
        for row in selected
    )
    return float(numerator / denominator)


def r8p_r5_small_case_dice(r5_summary):
    bins = dict(r5_summary.get('lesion_bin_mean_dice', {}))
    values = [
        float(bins[name]) for name in ('tiny', 'small')
        if name in bins and np.isfinite(float(bins[name]))
    ]
    if not values:
        raise RuntimeError('R5 baseline has no finite tiny/small case Dice.')
    return float(np.mean(values))


def r8p_checkpoint_evaluation(metrics, baseline):
    required = [
        'positive_mean_dice', 'positive_global_dice',
        'small_case_mean_dice', 'small_component_recall', 'lesion_f1',
    ]
    if any(name not in metrics for name in required):
        raise RuntimeError('R8 checkpoint metrics are incomplete.')
    if not all(np.isfinite(float(metrics[name])) for name in required):
        raise RuntimeError('R8 checkpoint metrics are non-finite.')
    guards = {
        'mean_dice_floor': float(metrics['positive_mean_dice'])
        >= float(baseline['mean_dice_floor']),
        'global_dice_floor': float(metrics['positive_global_dice'])
        >= float(baseline['global_dice_floor']),
        'small_component_improved': float(metrics['small_component_recall'])
        > float(baseline['small_component_recall']),
    }
    key = (
        float(metrics['positive_mean_dice']),
        float(metrics['small_case_mean_dice']),
        float(metrics['small_component_recall']),
        float(metrics['lesion_f1']),
        float(metrics['positive_global_dice']),
    )
    return all(guards.values()), guards, key


def run_r8p_fold0_pilot_preflight():
    if r8p_existing_preflight_complete():
        gate = load_json(R8P_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 FOLD-0 PILOT PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Pilot version: {gate["pilot_version"]}')
        print(f'Preflight gate: {R8P_GATE_PATH}')
        print('No source checkpoint, model checkpoint, or artifact was rewritten.')
        return

    r8_gate = load_json(R8_GATE_PATH)
    r8_policy = load_json(R8_TRAINING_POLICY_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_baseline = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
    r5_summary = dict(r5_baseline['validation_summary'])
    r5_small_component_recall = r8p_r5_small_component_recall(r5_baseline)
    r5_small_case_dice = r8p_r5_small_case_dice(r5_summary)
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r8_sha = file_sha256(R8_GATE_PATH)
    upstream_artifacts_match = all(
        (R8_PREFLIGHT_ROOT / name).exists()
        and file_sha256(R8_PREFLIGHT_ROOT / name) == expected
        for name, expected in r8_gate.get('artifact_sha256', {}).items()
    )
    upstream_checks = {
        'r8_architecture_preflight_passed': (
            r8_gate.get('preflight_passed') is True
        ),
        'r8_all_architecture_checks_passed': (
            len(r8_gate.get('checks', {})) > 0
            and all(bool(value) for value in r8_gate['checks'].values())
        ),
        'r8_preflight_version_matches': (
            r8_gate.get('preflight_version') == R8_PREFLIGHT_VERSION
        ),
        'r8_code_revision_matches': (
            r8_gate.get('code_revision') == R8_CODE_REVISION
        ),
        'r8_artifact_hashes_match': upstream_artifacts_match,
        'r8_pilot_implementation_allowed': (
            r8_gate.get('r8_pilot_implementation_allowed') is True
        ),
        'r8_pilot_was_training_locked': (
            r8_gate.get('r8_pilot_training_authorized') is False
        ),
        'full_fivefold_was_locked': (
            r8_gate.get('full_fivefold_training_authorized') is False
        ),
        'r8_parameter_count_matches': (
            int(r8_gate.get('total_parameters', -1))
            == R8P_EXPECTED_PARAMETERS
        ),
        'r8_warm_start_is_r5': (
            str(r8_gate.get('r5_checkpoint_path'))
            == str(R5_BEST_CHECKPOINT_PATH)
        ),
        'r8_warm_start_hash_matches': (
            r8_gate.get('r5_checkpoint_sha256') == r5_sha_before
        ),
        'r7_weights_remain_excluded': (
            r8_gate.get('r7_checkpoint_used_for_weights') is False
        ),
        'r5_result_complete': r5_result.get('pilot_complete') is True,
        'r5_baseline_metrics_finite': bool(np.isfinite([
            float(r5_summary['positive_mean_dice']),
            float(r5_summary['positive_global_dice']),
            r5_small_case_dice, r5_small_component_recall,
        ]).all()),
        'fold0_partition_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_and_test_counts_match': (
            len(locked_validation_df) == 23 and len(sealed_test_df) == 23
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 PILOT PREFLIGHT UPSTREAM FAILURE. Do not continue. '
            'Failed: ' + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R8 FOLD-0 PILOT LOADER/LOSS/TRAINING PREFLIGHT STARTING')
    print('=' * 80)
    print('Warm start:                         r5 best EMA (epoch 23)')
    print('R7 weights used:                    False')
    print('Target access:                      fold-0 fit smoke cases only')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer steps:                    0')

    dataset = Trial8R8Fold0Dataset()
    selected_indices = []
    for tier in R8_TIER_QUOTAS:
        selected_indices.append(next(
            index for index, entry in enumerate(dataset.schedule)
            if str(entry['sample_type']) == 'positive'
            and str(entry['component_tier']) == tier
        ))
    selected_indices.extend([
        next(index for index, entry in enumerate(dataset.schedule)
             if str(entry['sample_type']) == 'topology_negative'),
        next(index for index, entry in enumerate(dataset.schedule)
             if str(entry['sample_type']) == 'brain_negative'),
    ])
    loader_rows = []
    positive_sample = None
    negative_sample = None
    for index in selected_indices:
        sample = dataset[index]
        float_arrays = [
            np.asarray(value) for value in sample.values()
            if isinstance(value, np.ndarray)
            and np.issubdtype(np.asarray(value).dtype, np.floating)
        ]
        row = {
            'subject_id': sample['subject_id'],
            'sample_type': sample['sample_type'],
            'component_tier': sample['component_tier'],
            'atomic_component_id': int(sample['atomic_component_id']),
            'target_soft_support_voxels': int(
                (sample['target_soft'] > 0).sum()
            ),
            'native_component_voxels': int(
                sample['native_component_target'].sum()
            ),
            'native_negative_annulus_voxels': int(
                sample['native_component_negative_annulus'].sum()
            ),
            'native_component_valid': bool(
                sample['native_component_valid']
            ),
            'component_retained_fraction': float(
                sample['component_retained_fraction']
            ),
            'all_float_arrays_finite': all(
                np.isfinite(value).all() for value in float_arrays
            ),
            'target_arrays_read_split': 'fold0_fit',
            'fold0_heldout_target_read': False,
            'locked_validation_target_read': False,
            'test_target_read': False,
        }
        loader_rows.append(row)
        if (
            positive_sample is None
            and sample['sample_type'] == 'positive'
            and sample['component_tier'] == 'micro'
        ):
            positive_sample = sample
        if negative_sample is None and sample['sample_type'] == 'brain_negative':
            negative_sample = sample
    if positive_sample is None or negative_sample is None:
        raise RuntimeError('R8 smoke samples were not resolved.')
    loader_frame = pd.DataFrame(loader_rows)
    atomic_write_csv(loader_frame, R8P_LOADER_AUDIT_PATH)

    positive_batch = torch.utils.data.default_collate([positive_sample])
    negative_batch = torch.utils.data.default_collate([negative_sample])
    expected_shapes = {
        'image': (1, 15, *LOCAL_PATCH_SIZE),
        'global_image': (1, 15, *GLOBAL_INPUT_SHAPE),
        'target_binary': (1, 1, *LOCAL_PATCH_SIZE),
        'target_soft': (1, 1, *LOCAL_PATCH_SIZE),
        'global_target': (1, 1, *GLOBAL_INPUT_SHAPE),
        'native_component_target': (1, 1, *LOCAL_PATCH_SIZE),
        'native_component_negative_annulus': (1, 1, *LOCAL_PATCH_SIZE),
        'local_origin_in_canvas': (1, 3),
    }
    batch_rows = []
    for name, expected in expected_shapes.items():
        observed = tuple(positive_batch[name].shape)
        batch_rows.append({
            'tensor': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(
                positive_batch[name].float()
            ).all().item()),
        })
    batch_frame = pd.DataFrame(batch_rows)
    atomic_write_csv(batch_frame, R8P_BATCH_AUDIT_PATH)

    random.seed(R8P_SEED)
    np.random.seed(R8P_SEED)
    torch.manual_seed(R8P_SEED)
    torch.cuda.manual_seed_all(R8P_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda':
        raise RuntimeError('Trial8 R8 fold-0 pilot preflight requires CUDA.')
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(checkpoint['ema_model_state'])
    del checkpoint
    total_parameters = sum(
        parameter.numel() for parameter in model.parameters()
    )
    parameter_digest_before = r8p_parameter_digest(model)
    optimizer_groups = r8p_optimizer_groups(model)
    r8p_apply_stage(model, 1)
    optimizer = torch.optim.AdamW(
        optimizer_groups, betas=(0.9, 0.99), eps=1e-8
    )
    optimizer_rows = []
    for group in optimizer.param_groups:
        optimizer_rows.append({
            'group_name': str(group['group_name']),
            'parameters': int(sum(
                parameter.numel() for parameter in group['params']
            )),
            'initial_lr': float(group['lr']),
            'weight_decay': float(group['weight_decay']),
            'optimizer_state_entries': int(sum(
                parameter in optimizer.state for parameter in group['params']
            )),
        })
    optimizer_frame = pd.DataFrame(optimizer_rows)
    atomic_write_csv(optimizer_frame, R8P_OPTIMIZER_AUDIT_PATH)

    stage_rows = []
    for epoch in (1, 4, 5, 12, 13, 40):
        stage, rates = r8p_apply_stage(model, epoch, optimizer)
        stage_rows.append({
            'epoch': epoch,
            'stage': stage,
            'trainable_parameters': int(sum(
                parameter.numel() for parameter in model.parameters()
                if parameter.requires_grad
            )),
            'base_trainable_parameters': int(sum(
                parameter.numel() for parameter in model.base.parameters()
                if parameter.requires_grad
            )),
            'satellite_head_trainable_parameters': int(sum(
                parameter.numel()
                for parameter in model.satellite_head.parameters()
                if parameter.requires_grad
            )),
            'satellite_gate_trainable_parameters': int(sum(
                parameter.numel()
                for parameter in model.satellite_gate.parameters()
                if parameter.requires_grad
            )),
            **{f'{name}_lr': value for name, value in rates.items()},
        })
    stage_frame = pd.DataFrame(stage_rows)
    atomic_write_csv(stage_frame, R8P_STAGE_AUDIT_PATH)
    lr_frame = stage_frame[[
        'epoch', 'stage', 'r5_base_lr', 'native_satellite_head_lr',
        'bounded_satellite_gate_lr',
    ]].copy()
    atomic_write_csv(lr_frame, R8P_LR_AUDIT_PATH)

    no_dropout = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    gradient_rows = []
    loss_rows = []
    gradient_probes = [
        (1, 'positive', positive_batch),
        (1, 'brain_negative', negative_batch),
        (5, 'positive', positive_batch),
        (13, 'positive', positive_batch),
    ]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    for epoch, sample_role, cpu_batch in gradient_probes:
        stage, rates = r8p_apply_stage(model, epoch, optimizer)
        model.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type='cuda', dtype=torch.bfloat16, enabled=True
        ):
            loss, parts, satellite_rows, output = r8p_compute_training_loss(
                model, cpu_batch, device, epoch,
                dropout_flags_override=no_dropout,
            )
        if not bool(torch.isfinite(loss).item()):
            raise RuntimeError('R8 stage loss smoke is non-finite.')
        loss.backward()
        group_modules = {
            'r5_base': model.base,
            'native_satellite_head': model.satellite_head,
            'bounded_satellite_gate': model.satellite_gate,
        }
        for group_name, module in group_modules.items():
            norm, observed, finite = r8_gradient_norm(module)
            gradient_rows.append({
                'epoch': epoch,
                'stage': stage,
                'sample_role': sample_role,
                'parameter_group': group_name,
                'gradient_norm': norm,
                'gradient_observed': observed,
                'gradient_finite': finite,
                'group_lr': float(rates[group_name]),
            })
        loss_record = {
            'epoch': epoch,
            'stage': stage,
            'sample_role': sample_role,
        }
        for name, value in parts.items():
            loss_record[name] = float(value.detach().float().cpu())
        if satellite_rows:
            loss_record.update({
                f'native_{name}': value
                for name, value in satellite_rows[0].items()
            })
        loss_rows.append(loss_record)
        del output, loss, parts
        model.zero_grad(set_to_none=True)
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_frame = pd.DataFrame(gradient_rows)
    loss_frame = pd.DataFrame(loss_rows)
    atomic_write_csv(gradient_frame, R8P_GRADIENT_AUDIT_PATH)
    atomic_write_csv(loss_frame, R8P_LOSS_AUDIT_PATH)

    r8p_apply_stage(model, 1, optimizer)
    model.eval()
    positive_device = pilot_to_device(positive_batch, device)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        direct = model(
            **{key: positive_device[key] for key in MODEL_INPUT_KEYS_V22},
            **no_dropout,
        )
        cache = pilot_global_cache(model.base, positive_device)
        cached = r8p_cached_local_from_global(
            model, positive_device, cache,
        )
    cache_rows = []
    for name in ('logits', 'base_logits', 'satellite_logits'):
        difference = float(torch.max(torch.abs(
            direct[name].float() - cached[name].float()
        )).cpu())
        cache_rows.append({
            'output': name,
            'maximum_absolute_direct_vs_cached_difference': difference,
            'within_bfloat16_tolerance': difference <= 0.02,
        })
    initial_gate_abs_max = float(
        direct['satellite_gate'].float().abs().max().cpu()
    )
    initial_final_base_difference = float(torch.max(torch.abs(
        direct['logits'].float() - direct['base_logits'].float()
    )).cpu())
    cache_frame = pd.DataFrame(cache_rows)
    atomic_write_csv(cache_frame, R8P_CACHE_AUDIT_PATH)
    del direct, cached, cache, positive_device

    baseline = {
        'r5_mean_dice': float(r5_summary['positive_mean_dice']),
        'r5_global_dice': float(r5_summary['positive_global_dice']),
        'mean_dice_floor': float(
            r8_policy['checkpoint_eligibility']['mean_dice_minimum']
        ),
        'global_dice_floor': float(
            r8_policy['checkpoint_eligibility']['global_dice_minimum']
        ),
        'small_case_mean_dice': r5_small_case_dice,
        'small_component_recall': r5_small_component_recall,
    }
    passing_metrics = {
        'positive_mean_dice': baseline['mean_dice_floor'] + 0.010,
        'positive_global_dice': baseline['global_dice_floor'] + 0.010,
        'small_case_mean_dice': baseline['small_case_mean_dice'] + 0.010,
        'small_component_recall': baseline['small_component_recall'] + 0.010,
        'lesion_f1': 0.20,
        'oracle_volume_budgeted_recall': 0.99,
    }
    mean_failure = dict(passing_metrics)
    mean_failure['positive_mean_dice'] = baseline['mean_dice_floor'] - 0.001
    small_failure = dict(passing_metrics)
    small_failure['small_component_recall'] = baseline[
        'small_component_recall'
    ]
    oracle_variant = dict(passing_metrics)
    oracle_variant['oracle_volume_budgeted_recall'] = 0.0
    selection_tests = []
    for name, metrics in [
        ('eligible_improvement', passing_metrics),
        ('mean_dice_regression', mean_failure),
        ('no_small_component_improvement', small_failure),
        ('oracle_metric_changed_only', oracle_variant),
    ]:
        eligible, guards, key = r8p_checkpoint_evaluation(metrics, baseline)
        selection_tests.append({
            'scenario': name,
            'eligible': eligible,
            'selection_key': str(key),
            **guards,
        })
    selection_frame = pd.DataFrame(selection_tests)
    atomic_write_csv(selection_frame, R8P_SELECTION_AUDIT_PATH)

    parameter_digest_after = r8p_parameter_digest(model)
    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    expected_trainable = {
        1: 14_481,
        4: 14_481,
        5: 14_706,
        12: 14_706,
        13: R8P_EXPECTED_PARAMETERS,
        40: R8P_EXPECTED_PARAMETERS,
    }

    def gradient_entry(epoch, role, group):
        selected = gradient_frame[
            (gradient_frame['epoch'] == epoch)
            & (gradient_frame['sample_role'] == role)
            & (gradient_frame['parameter_group'] == group)
        ]
        if len(selected) != 1:
            raise RuntimeError('R8 gradient audit row is not unique.')
        return selected.iloc[0]

    policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'warm_start': {
            'source': 'r5_best_ema_epoch_23',
            'checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'checkpoint_sha256': r5_sha_before,
            'r7_weights_used': False,
            'parent_checkpoint_migration_allowed': False,
        },
        'epoch_schedule': {
            'maximum_epochs': R8P_MAX_EPOCHS,
            'minimum_epochs': R8P_MINIMUM_EPOCHS,
            'early_stopping_patience': R8P_EARLY_STOPPING_PATIENCE,
            'minimum_delta_for_patience_only': (
                R8P_EARLY_STOPPING_MINIMUM_DELTA
            ),
            'checkpoint_save_rule': 'every_strict_lexicographic_improvement',
            'metrics_printed_every_epoch': True,
        },
        'optimization_stages': r8_policy['optimization_stages'],
        'learning_rates': {
            'r5_base_peak': R8P_BASE_LR,
            'satellite_head_peak': R8P_HEAD_LR,
            'satellite_gate_peak': R8P_GATE_LR,
            'branch_local_two_epoch_warmup': True,
            'cosine_decay_to_fraction': R8P_MINIMUM_LR_FRACTION,
            'accumulation_steps': R8P_ACCUMULATION_STEPS,
            'gradient_clip_norm': R8P_GRADIENT_CLIP_NORM,
        },
        'loss': {
            'preserved_r5_main_and_auxiliary_objective': True,
            'coarse_proposal_whole_mask_only': True,
            'coarse_component_contrastive_disabled': True,
            'native_component_objective': (
                'positive_bce_plus_hard_negative_bce_plus_ranking_plus_'
                'absolute_positive_floor'
            ),
            'target_free_negative_objective': (
                'top4096_native_satellite_hard_background_softplus'
            ),
            'negative_background_weight': R8P_NEGATIVE_BACKGROUND_WEIGHT,
            'satellite_loss_coefficient': R8P_SATELLITE_LOSS_COEFFICIENT,
            'volume_case_weights': False,
            'topk_ohem_on_main_segmentation': False,
        },
        'validation': {
            'partition': 'fold0_heldout_development_only',
            'frequency': 'every_epoch',
            'local_threshold_grid': R8P_LOCAL_THRESHOLDS.tolist(),
            'proposal_threshold_grid': R8P_PROPOSAL_THRESHOLDS.tolist(),
            'sliding_window_overlap': R8P_VALIDATION_OVERLAP,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'per_epoch_diagnostics': [
                'mean_median_global_dice', 'lesion_precision_recall_f1',
                'lesion_volume_bin_dice', 'all_and_small_component_recall',
                'satellite_positive_and_hard_negative_logit_margin',
                'proposal_precision_ap_and_volume_ratio',
                'gate_distribution_and_thresholds',
            ],
            'every_five_epoch_diagnostics': [
                'satellite_ablation', 'global_ablation',
                'arterial_ablation', 'clinical_ablation',
                'r5_base_parameter_drift',
            ],
        },
        'checkpoint_eligibility': {
            **baseline,
            'small_component_improvement_required': True,
            'selection_order': [
                'positive_mean_dice', 'small_case_mean_dice',
                'small_component_recall', 'lesion_f1',
                'positive_global_dice',
            ],
            'oracle_metrics_used_for_selection': False,
            'thresholds_saved_with_checkpoint': True,
            'if_no_r8_epoch_is_eligible': (
                'declare_directional_failure_and_retain_r5_best_as_fallback'
            ),
        },
        'output_paths': {
            'pilot_root': str(R8P_PILOT_ROOT),
            'best_checkpoint': str(R8P_BEST_CHECKPOINT_PATH),
            'last_checkpoint': str(R8P_LAST_CHECKPOINT_PATH),
            'history': str(R8P_HISTORY_PATH),
            'result_gate': str(R8P_RESULT_GATE_PATH),
        },
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(policy, R8P_POLICY_PATH)
    policy_sha256 = file_sha256(R8P_POLICY_PATH)
    resume_contract = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'pilot_policy_sha256': policy_sha256,
        'accepted_resume_source': 'same_r8_pilot_last_checkpoint_only',
        'fresh_start_source': 'r5_best_ema_plus_deterministic_r8_initialization',
        'r7_checkpoint_migration_allowed': False,
        'r5_checkpoint_migration_after_epoch_zero_allowed': False,
        'deployment_fallback_if_no_eligible_r8_checkpoint': str(
            R5_BEST_CHECKPOINT_PATH
        ),
        'required_checkpoint_fields': [
            'pilot_version', 'code_revision', 'pilot_policy_sha256',
            'epoch', 'model_state', 'ema_model_state', 'optimizer_state',
            'rng_state', 'best_selection_key', 'best_thresholds',
            'optimizer_steps',
        ],
        'rng_restore_policy': (
            'normalize_cpu_and_cuda_rng_tensors_to_contiguous_cpu_uint8'
        ),
        'checkpoint_write_policy': (
            'atomic_last_each_epoch_and_atomic_best_on_every_strict_eligible_'
            'lexicographic_improvement'
        ),
        'retroactive_best_reconstruction': False,
        'mismatched_config_action': 'refuse_resume_without_overwrite',
    }
    atomic_write_json(resume_contract, R8P_RESUME_PATH)

    stage_counts_match = all(
        int(row['trainable_parameters']) == expected_trainable[int(row['epoch'])]
        for row in stage_rows
    )
    positive_loss_row = loss_frame[
        (loss_frame['epoch'] == 1)
        & (loss_frame['sample_role'] == 'positive')
    ].iloc[0]
    negative_loss_row = loss_frame[
        (loss_frame['epoch'] == 1)
        & (loss_frame['sample_role'] == 'brain_negative')
    ].iloc[0]
    oracle_a = selection_frame[
        selection_frame['scenario'] == 'eligible_improvement'
    ].iloc[0]
    oracle_b = selection_frame[
        selection_frame['scenario'] == 'oracle_metric_changed_only'
    ].iloc[0]
    checks = {
        **upstream_checks,
        'dataset_has_168_samples': len(dataset) == 168,
        'dataset_has_112_positive_samples': sum(
            str(item['sample_type']) == 'positive'
            for item in dataset.schedule
        ) == 112,
        'dataset_has_28_topology_negatives': sum(
            str(item['sample_type']) == 'topology_negative'
            for item in dataset.schedule
        ) == 28,
        'dataset_has_28_brain_negatives': sum(
            str(item['sample_type']) == 'brain_negative'
            for item in dataset.schedule
        ) == 28,
        'loader_smoke_covers_all_four_component_tiers': (
            set(loader_frame[
                loader_frame['sample_type'] == 'positive'
            ]['component_tier']) == set(R8_TIER_QUOTAS)
        ),
        'loader_smoke_covers_both_negative_roles': (
            {'topology_negative', 'brain_negative'}.issubset(
                set(loader_frame['sample_type'])
            )
        ),
        'positive_loader_cases_have_native_targets': bool((
            loader_frame[loader_frame['sample_type'] == 'positive'][
                'native_component_voxels'
            ] > 0
        ).all()),
        'positive_loader_cases_have_native_annuli': bool((
            loader_frame[loader_frame['sample_type'] == 'positive'][
                'native_negative_annulus_voxels'
            ] > 0
        ).all()),
        'negative_loader_cases_are_target_free': bool((
            loader_frame[loader_frame['sample_type'] != 'positive'][
                'target_soft_support_voxels'
            ] == 0
        ).all()),
        'all_loader_float_arrays_are_finite': bool(
            loader_frame['all_float_arrays_finite'].all()
        ),
        'all_batch_shapes_match': bool(batch_frame['shape_matches'].all()),
        'all_batch_tensors_are_finite': bool(batch_frame['finite'].all()),
        'total_parameter_count_matches': (
            total_parameters == R8P_EXPECTED_PARAMETERS
        ),
        'optimizer_groups_cover_model_exactly': (
            int(optimizer_frame['parameters'].sum()) == total_parameters
        ),
        'optimizer_groups_are_disjoint': True,
        'optimizer_state_is_empty': len(optimizer.state) == 0,
        'optimizer_steps_are_zero': True,
        'stage_trainable_counts_match': stage_counts_match,
        'base_lr_is_zero_before_epoch13': bool((
            lr_frame[lr_frame['epoch'] < 13]['r5_base_lr'] == 0.0
        ).all()),
        'gate_lr_is_zero_before_epoch5': bool((
            lr_frame[lr_frame['epoch'] < 5][
                'bounded_satellite_gate_lr'
            ] == 0.0
        ).all()),
        'head_lr_is_positive_from_epoch1': bool((
            lr_frame['native_satellite_head_lr'] > 0.0
        ).all()),
        'all_loss_smokes_are_finite': bool(np.isfinite(
            loss_frame[[
                'total_loss', 'preserved_objective', 'main_loss',
                'proposal_whole_mask_loss', 'presence_loss',
                'volume_loss', 'arterial_loss',
                'satellite_positive_loss',
                'satellite_negative_background_loss',
                'satellite_objective', 'satellite_loss_coefficient',
                'hard_background_voxels', 'native_components_in_batch',
            ]].to_numpy(dtype=np.float64)
        ).all()),
        'positive_native_absolute_floor_is_active': (
            float(positive_loss_row['native_absolute_positive_floor_loss'])
            > 0.0
        ),
        'negative_native_background_loss_is_active': (
            float(negative_loss_row['satellite_negative_background_loss'])
            > 0.0
            and float(negative_loss_row['hard_background_voxels'])
            == R8P_NEGATIVE_BACKGROUND_TOPK
        ),
        'coarse_component_contrastive_is_absent': (
            'local_contrastive_loss' not in loss_frame.columns
        ),
        'stage1_positive_head_gradient_present': bool(
            gradient_entry(1, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage1_negative_head_gradient_present': bool(
            gradient_entry(1, 'brain_negative', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage1_base_gradient_absent': not bool(
            gradient_entry(1, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage1_gate_gradient_absent': not bool(
            gradient_entry(1, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'stage2_head_gradient_present': bool(
            gradient_entry(5, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage2_gate_gradient_present': bool(
            gradient_entry(5, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'stage2_base_gradient_absent': not bool(
            gradient_entry(5, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage3_base_gradient_present': bool(
            gradient_entry(13, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage3_head_gradient_present': bool(
            gradient_entry(13, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage3_gate_gradient_present': bool(
            gradient_entry(13, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'all_observed_gradients_are_finite': bool(
            gradient_frame['gradient_finite'].all()
        ),
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'cached_validation_adapter_matches_direct_forward': bool(
            cache_frame['within_bfloat16_tolerance'].all()
        ),
        'actual_pilot_model_gate_is_zero_at_initialization': (
            initial_gate_abs_max == 0.0
        ),
        'actual_pilot_model_initial_output_equals_r5_base': (
            initial_final_base_difference == 0.0
        ),
        'eligible_checkpoint_scenario_passes': bool(
            selection_frame[
                selection_frame['scenario'] == 'eligible_improvement'
            ]['eligible'].iloc[0]
        ),
        'mean_dice_regression_is_rejected': not bool(
            selection_frame[
                selection_frame['scenario'] == 'mean_dice_regression'
            ]['eligible'].iloc[0]
        ),
        'missing_small_component_improvement_is_rejected': not bool(
            selection_frame[
                selection_frame['scenario']
                == 'no_small_component_improvement'
            ]['eligible'].iloc[0]
        ),
        'oracle_metric_cannot_change_selection': (
            bool(oracle_a['eligible']) == bool(oracle_b['eligible'])
            and str(oracle_a['selection_key']) == str(oracle_b['selection_key'])
        ),
        'checkpoint_paths_are_isolated_from_r5_and_r7': (
            R8P_BEST_CHECKPOINT_PATH not in {
                R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH
            }
            and R8P_LAST_CHECKPOINT_PATH != PILOT_LAST_CHECKPOINT_PATH
        ),
        'resume_contract_hashes_exact_pilot_policy': (
            resume_contract['pilot_policy_sha256']
            == file_sha256(R8P_POLICY_PATH)
        ),
        'no_r8_checkpoint_was_written': (
            not R8P_BEST_CHECKPOINT_PATH.exists()
            and not R8P_LAST_CHECKPOINT_PATH.exists()
        ),
        'model_parameters_unchanged_by_backward_smokes': (
            parameter_digest_before == parameter_digest_after
        ),
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'training_not_performed': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in checks.items()
    ])
    atomic_write_csv(checks_frame, R8P_CHECKS_PATH)
    preflight_passed = all(checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R8P_OUTPUT_ARTIFACTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8P_PREFLIGHT_VERSION,
        'code_revision': R8P_CODE_REVISION,
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r8_architecture_gate_path': str(R8_GATE_PATH),
        'r8_architecture_gate_sha256': r8_sha,
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_weights_used': False,
        'total_parameters': total_parameters,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'fit_targets_read': len(set(loader_frame['subject_id'])),
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': True,
        'optimizer_steps': 0,
        'training_performed': False,
        'r8_fold0_pilot_training_authorized': bool(preflight_passed),
        'full_fivefold_training_authorized': False,
        'pilot_policy_path': str(R8P_POLICY_PATH),
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'resume_contract_path': str(R8P_RESUME_PATH),
        'checks': checks,
        'artifact_sha256': artifact_hashes,
    }
    atomic_write_json(gate, R8P_GATE_PATH)

    del model, optimizer, positive_batch, negative_batch
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 FOLD-0 PILOT PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR8 loader smoke cases:')
    display(loader_frame)
    print('\nR8 staged trainability and learning rates:')
    display(stage_frame)
    print('\nR8 real-patch loss smoke:')
    display(loss_frame)
    print('\nR8 staged gradient audit:')
    display(gradient_frame)
    print('\nR8 checkpoint-policy tests:')
    display(selection_frame)
    print('\nR8 fold-0 pilot preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R8 FOLD-0 PILOT LOADER/LOSS/TRAINING PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Pilot version:                       {R8P_PILOT_VERSION}')
    print(f'Fit / held-out subjects:             {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Total parameters:                    {total_parameters:,}')
    print(f'Peak gradient memory GB:             {peak_gradient_memory_gb:.3f}')
    print('Stage 1 negative patches train head: True')
    print('Coarse component contrastive loss:   disabled')
    print('Native absolute calibration loss:    enabled')
    print('R5 mean/global Pareto floors:         enforced')
    print('Oracle metrics select checkpoints:   False')
    print('Metrics printed:                      every epoch')
    print('Fold-0 held-out targets read:         False')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Optimizer steps:                      0')
    print('Training performed:                   False')
    print('R8 fold-0 pilot training authorized:  True')
    print('Full five-fold training authorized:   False')
    print(f'Pilot policy:                         {R8P_POLICY_PATH}')
    print(f'Resume contract:                      {R8P_RESUME_PATH}')
    print(f'Preflight gate:                       {R8P_GATE_PATH}')
    print(
        '\nNext: build and run the isolated fold-0 R8 pilot training cell '
        'from this frozen gate; do not launch five folds.'
    )


run_r8p_fold0_pilot_preflight()











R8T_IMPLEMENTATION_REVISION = 'r2_strict_r5_ema_pareto_epoch_diagnostics'
R8T_VALIDATION_ROOT = R8P_PILOT_ROOT / 'validation'
R8T_DIAGNOSTIC_ROOT = R8P_PILOT_ROOT / 'diagnostics'
R8T_STATE_PATH = R8P_PILOT_ROOT / 'pilot_state.json'
R8T_BRANCH_HISTORY_PATH = R8P_PILOT_ROOT / 'branch_reliance_history.csv'
R8T_BASE_DRIFT_HISTORY_PATH = R8P_PILOT_ROOT / 'r5_base_drift_history.csv'
R8T_SATELLITE_HISTORY_PATH = R8P_PILOT_ROOT / 'satellite_calibration_history.csv'
R8T_FINAL_SUMMARY_PATH = R8P_PILOT_ROOT / 'final_pilot_summary.json'

R8T_VALIDATION_CAPTURE = []
R8T_SMALL_TIERS = {
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
}
R8T_NO_DROPOUT = {
    'force_atlas_dropout': False,
    'force_vascular_dropout': False,
    'force_occlusion_dropout': False,
    'force_clinical_dropout': False,
    'force_global_dropout': False,
    'force_arterial_dropout': False,
}


def r8t_existing_result_complete():
    if not R8P_RESULT_GATE_PATH.exists():
        return False
    result = load_json(R8P_RESULT_GATE_PATH)
    if not (
        result.get('pilot_complete') is True
        and result.get('pilot_version') == R8P_PILOT_VERSION
        and result.get('code_revision') == R8P_PILOT_CODE_REVISION
        and result.get('implementation_revision')
        == R8T_IMPLEMENTATION_REVISION
        and result.get('pilot_policy_sha256')
        == file_sha256(R8P_POLICY_PATH)
        and result.get('preflight_gate_sha256')
        == file_sha256(R8P_GATE_PATH)
        and result.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and R8P_LAST_CHECKPOINT_PATH.exists()
        and result.get('last_checkpoint_sha256')
        == file_sha256(R8P_LAST_CHECKPOINT_PATH)
        and R8T_FINAL_SUMMARY_PATH.exists()
        and result.get('final_summary_sha256')
        == file_sha256(R8T_FINAL_SUMMARY_PATH)
        and R8P_HISTORY_PATH.exists()
    ):
        return False
    if bool(result.get('eligible_r8_checkpoint_found')):
        return (
            R8P_BEST_CHECKPOINT_PATH.exists()
            and result.get('best_checkpoint_sha256')
            == file_sha256(R8P_BEST_CHECKPOINT_PATH)
        )
    return not R8P_BEST_CHECKPOINT_PATH.exists()


def r8t_dropout_flags(epoch, batch_index):
    rng = np.random.default_rng(
        R8P_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    return pilot_dropout_flags(rng)


def r8t_ema_update(ema_model, model, optimizer_step):
    decay = min(
        EMA_MAX_DECAY,
        1.0 - 1.0 / float(int(optimizer_step) + 1),
    )
    with torch.no_grad():
        source = model.state_dict()
        destination = ema_model.state_dict()
        if source.keys() != destination.keys():
            raise RuntimeError('R8 EMA source/destination state schemas differ.')
        for name, value in destination.items():
            source_value = source[name].detach()
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source_value, alpha=1.0 - decay)
            else:
                value.copy_(source_value)


def r8t_small_component_recall(tier_rows):
    selected = [
        row for row in tier_rows
        if str(row.get('component_tier')) in R8T_SMALL_TIERS
    ]
    denominator = sum(int(row.get('components', 0)) for row in selected)
    if denominator <= 0:
        raise RuntimeError('R8 validation has no small components.')
    numerator = sum(
        int(row.get('components', 0))
        * float(row.get('final_component_recall', 0.0))
        for row in selected
    )
    return float(numerator / denominator)


def r8t_component_satellite_records(satellite_logit, target, spacing):
    target = np.asarray(target, dtype=bool)
    satellite_logit = np.asarray(satellite_logit, dtype=np.float32)
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    for component_id, component_slice in enumerate(
        ndimage.find_objects(labels), start=1
    ):
        if component_slice is None:
            continue
        component = labels[component_slice] == component_id
        values = satellite_logit[component_slice][component]
        voxels = int(component.sum())
        records.append({
            'component_id': int(component_id),
            'component_tier': pilot_component_tier(
                voxels * voxel_volume_ml, voxels
            ),
            'component_voxels': voxels,
            'satellite_logit_mean': float(values.mean()),
            'satellite_logit_maximum': float(values.max(initial=-np.inf)),
            'satellite_probability_maximum': float(
                torch.sigmoid(torch.from_numpy(values)).max().item()
            ),
        })
    if len(records) != int(count):
        raise RuntimeError('R8 satellite component diagnostic lost a component.')
    return records


@torch.no_grad()
def r8t_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model.base, base_batch,
            force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )

    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(
            native_shape[axis], LOCAL_PATCH_SIZE[axis],
            overlap=R8P_VALIDATION_OVERLAP,
        )
        for axis in range(3)
    ]
    final_sum = np.zeros(native_shape, dtype=np.float32)
    base_sum = np.zeros(native_shape, dtype=np.float32)
    satellite_logit_sum = np.zeros(native_shape, dtype=np.float32)
    gate_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][axis] + native_start[axis])
                    for axis in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = r8p_cached_local_from_global(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                base_patch = torch.sigmoid(
                    output['base_logits'].float()
                )[0, 0].cpu().numpy()
                final_patch = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                if ablation == 'satellite':
                    final_patch = base_patch
                satellite_patch = output[
                    'satellite_logits'
                ].float()[0, 0].cpu().numpy()
                gate_value = float(
                    output['satellite_gate'].float().reshape(-1)[0].cpu()
                )
                destination = tuple(
                    slice(native_start[axis], min(
                        native_start[axis] + LOCAL_PATCH_SIZE[axis],
                        native_shape[axis],
                    ))
                    for axis in range(3)
                )
                copied_shape = tuple(
                    item.stop - item.start for item in destination
                )
                source = tuple(
                    slice(0, copied_shape[axis]) for axis in range(3)
                )
                weight = PILOT_WINDOW_WEIGHT[source]
                final_sum[destination] += final_patch[source] * weight
                base_sum[destination] += base_patch[source] * weight
                satellite_logit_sum[destination] += (
                    satellite_patch[source] * weight
                )
                gate_sum[destination] += gate_value * weight
                weight_sum[destination] += weight
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete R8 inference coverage."
        )
    probability = final_sum / weight_sum
    base_probability = base_sum / weight_sum
    satellite_logit = satellite_logit_sum / weight_sum
    satellite_gate_map = gate_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(
            case['pad_start'][axis],
            case['pad_start'][axis] + native_shape[axis],
        )
        for axis in range(3)
    )
    proposal = proposal_canvas[native_slices]

    target = case['target']
    hard_background = satellite_logit[~target]
    hard_count = min(R8P_NEGATIVE_BACKGROUND_TOPK, hard_background.size)
    if hard_count > 0:
        hard_negative_mean = float(np.partition(
            hard_background, hard_background.size - hard_count
        )[-hard_count:].mean())
    else:
        hard_negative_mean = float('nan')
    positive_mean = (
        float(satellite_logit[target].mean())
        if bool(target.any()) else float('nan')
    )
    component_records = r8t_component_satellite_records(
        satellite_logit, target, case['spacing']
    )
    if ablation == 'none':
        R8T_VALIDATION_CAPTURE.append({
            'subject_id': case['subject_id'],
            'satellite_positive_logit_mean': positive_mean,
            'satellite_hard_negative_logit_mean': hard_negative_mean,
            'satellite_positive_minus_hard_negative_margin': (
                positive_mean - hard_negative_mean
                if np.isfinite(positive_mean) and np.isfinite(hard_negative_mean)
                else float('nan')
            ),
            'satellite_gate_mean': float(satellite_gate_map.mean()),
            'satellite_gate_abs_mean': float(
                np.abs(satellite_gate_map).mean()
            ),
            'satellite_gate_minimum': float(satellite_gate_map.min()),
            'satellite_gate_maximum': float(satellite_gate_map.max()),
            'base_to_final_mean_absolute_delta': float(
                np.abs(probability - base_probability).mean()
            ),
            'base_to_final_target_mean_absolute_delta': (
                float(np.abs(
                    probability[target] - base_probability[target]
                ).mean()) if bool(target.any()) else float('nan')
            ),
            'component_records': component_records,
        })
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': target, 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def r8t_satellite_validation_summary(epoch):
    rows = []
    component_rows = []
    for item in R8T_VALIDATION_CAPTURE:
        flat = {
            key: value for key, value in item.items()
            if key != 'component_records'
        }
        flat['epoch'] = int(epoch)
        rows.append(flat)
        for component in item['component_records']:
            component_rows.append({
                'epoch': int(epoch),
                'subject_id': item['subject_id'],
                **component,
            })
    frame = pd.DataFrame(rows)
    component_frame = pd.DataFrame(component_rows)
    epoch_root = R8T_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    atomic_write_csv(frame, epoch_root / 'satellite_case_metrics.csv')
    atomic_write_csv(
        component_frame, epoch_root / 'satellite_component_metrics.csv'
    )
    finite_margins = (
        frame['satellite_positive_minus_hard_negative_margin']
        .replace([np.inf, -np.inf], np.nan).dropna()
        if len(frame) else pd.Series(dtype=float)
    )
    small_components = (
        component_frame[
            component_frame['component_tier'].astype(str).isin(
                R8T_SMALL_TIERS
            )
        ] if len(component_frame) else component_frame
    )
    return {
        'satellite_positive_minus_hard_negative_logit_margin': (
            float(finite_margins.mean()) if len(finite_margins) else 0.0
        ),
        'satellite_gate_mean': (
            float(frame['satellite_gate_mean'].mean()) if len(frame) else 0.0
        ),
        'satellite_gate_abs_mean': (
            float(frame['satellite_gate_abs_mean'].mean())
            if len(frame) else 0.0
        ),
        'satellite_gate_minimum': (
            float(frame['satellite_gate_minimum'].min()) if len(frame) else 0.0
        ),
        'satellite_gate_maximum': (
            float(frame['satellite_gate_maximum'].max()) if len(frame) else 0.0
        ),
        'base_to_final_mean_absolute_delta': (
            float(frame['base_to_final_mean_absolute_delta'].mean())
            if len(frame) else 0.0
        ),
        'base_to_final_target_mean_absolute_delta': float(
            frame['base_to_final_target_mean_absolute_delta']
            .replace([np.inf, -np.inf], np.nan).dropna().mean()
        ) if len(frame) else 0.0,
        'small_satellite_component_representation_failure_fraction': (
            float(np.mean(
                small_components['satellite_probability_maximum'] < 0.10
            )) if len(small_components) else 0.0
        ),
        'small_satellite_component_probability_maximum_mean': (
            float(small_components[
                'satellite_probability_maximum'
            ].mean()) if len(small_components) else 0.0
        ),
        'satellite_components_audited': int(len(component_frame)),
        'satellite_small_components_audited': int(len(small_components)),
    }


@torch.no_grad()
def r8t_validate(model, epoch, device):
    global PILOT_VALIDATION_ROOT, LOCAL_THRESHOLDS, PROPOSAL_THRESHOLDS
    global pilot_infer_case, R8T_VALIDATION_CAPTURE
    original_root = PILOT_VALIDATION_ROOT
    original_local_thresholds = LOCAL_THRESHOLDS
    original_proposal_thresholds = PROPOSAL_THRESHOLDS
    original_infer_case = pilot_infer_case
    R8T_VALIDATION_CAPTURE = []
    try:
        PILOT_VALIDATION_ROOT = R8T_VALIDATION_ROOT
        LOCAL_THRESHOLDS = R8P_LOCAL_THRESHOLDS
        PROPOSAL_THRESHOLDS = R8P_PROPOSAL_THRESHOLDS
        pilot_infer_case = r8t_infer_case
        summary, tier_rows = pilot_validate(model, epoch, device)
    finally:
        PILOT_VALIDATION_ROOT = original_root
        LOCAL_THRESHOLDS = original_local_thresholds
        PROPOSAL_THRESHOLDS = original_proposal_thresholds
        pilot_infer_case = original_infer_case
    satellite_summary = r8t_satellite_validation_summary(epoch)
    summary.update(satellite_summary)
    summary['small_component_recall'] = r8t_small_component_recall(tier_rows)
    epoch_root = R8T_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    return summary, tier_rows


@torch.no_grad()
def r8t_branch_probe(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['satellite', 'global', 'arterial', 'clinical']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8),
            GLOBAL_CANVAS_SHAPE, (0, 1, 2),
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_arterial = ablation == 'arterial'
            force_clinical = ablation == 'clinical'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model.base, batch,
                    force_arterial_dropout=force_arterial,
                    force_clinical_dropout=force_clinical,
                )
                output = r8p_cached_local_from_global(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_clinical_dropout=force_clinical,
                )
            logits = (
                output['base_logits']
                if ablation == 'satellite' else output['logits']
            )
            return (
                torch.sigmoid(logits.float())[0, 0].cpu().numpy(),
                torch.sigmoid(
                    cache['proposal_logits'].float()
                )[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            rows.append({
                'epoch': int(epoch),
                'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.abs(baseline_local - ablated_local).mean()
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.abs(baseline_local - ablated_local).max()
                ),
                'mean_absolute_target_probability_delta': (
                    float(np.abs(
                        baseline_local[local_target]
                        - ablated_local[local_target]
                    ).mean()) if bool(local_target.any()) else float('nan')
                ),
                'mean_absolute_proposal_probability_delta': float(
                    np.abs(baseline_proposal - ablated_proposal).mean()
                ),
            })
    frame = pd.DataFrame(rows)
    R8T_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(
        frame,
        R8T_DIAGNOSTIC_ROOT
        / f'epoch_{int(epoch):03d}_branch_reliance.csv',
    )
    if R8T_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(R8T_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, R8T_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()


def r8t_base_drift(model, r5_reference_state, epoch):
    squared_difference = 0.0
    squared_reference = 0.0
    maximum_absolute_difference = 0.0
    compared = 0
    for name, parameter in model.base.named_parameters():
        if name not in r5_reference_state:
            raise RuntimeError(f'R5 reference lacks base parameter {name}.')
        observed = parameter.detach().float().cpu()
        reference = r5_reference_state[name].detach().float().cpu()
        difference = observed - reference
        squared_difference += float(torch.sum(difference * difference))
        squared_reference += float(torch.sum(reference * reference))
        maximum_absolute_difference = max(
            maximum_absolute_difference,
            float(difference.abs().max().item()),
        )
        compared += int(parameter.numel())
    row = {
        'epoch': int(epoch),
        'parameters_compared': int(compared),
        'relative_l2_drift': float(math.sqrt(
            squared_difference / max(squared_reference, 1e-24)
        )),
        'maximum_absolute_parameter_difference': float(
            maximum_absolute_difference
        ),
    }
    frame = pd.DataFrame([row])
    if R8T_BASE_DRIFT_HISTORY_PATH.exists():
        prior = pd.read_csv(R8T_BASE_DRIFT_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, R8T_BASE_DRIFT_HISTORY_PATH)
    return row


def r8t_recommendation(best_summary, baseline, eligible):
    if not eligible:
        return (
            'Retain r5. No R8 epoch satisfied all deployment-metric guards; '
            'inspect the per-tier and satellite-calibration audits.'
        )
    if float(best_summary[
        'satellite_positive_minus_hard_negative_logit_margin'
    ]) <= 0.0:
        return (
            'Keep the R5 base and revise the satellite absolute-calibration '
            'objective before any fold replication.'
        )
    if float(best_summary['small_component_recall']) <= float(
        baseline['small_component_recall']
    ):
        return (
            'Satellite features separate positives but are not converting to '
            'detections; revise the bounded fusion/threshold contract.'
        )
    if float(best_summary['positive_mean_dice']) < float(
        baseline['r5_mean_dice']
    ):
        return (
            'Small-component recall improved at a mean-Dice cost; reduce the '
            'gate or joint-base learning rate before replication.'
        )
    return (
        'R8 is directionally successful on fold 0. Build a separate '
        'fold-replication preflight; do not open the sealed test set yet.'
    )


def run_r8_fold0_pilot_training():
    if r8t_existing_result_complete():
        completed = load_json(R8P_RESULT_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 FOLD-0 PILOT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Best eligible epoch:                {completed.get("best_epoch")}')
        print(f'Directional success:                '
              f'{completed.get("directional_success")}')
        print(f'Result gate:                        {R8P_RESULT_GATE_PATH}')
        print('No model checkpoint or training artifact was rewritten.')
        return

    preflight_gate = load_json(R8P_GATE_PATH)
    r8_policy = load_json(R8P_POLICY_PATH)
    resume_contract = load_json(R8P_RESUME_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_baseline = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
    r5_summary = dict(r5_baseline['validation_summary'])
    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    preflight_checks = {
        'r8_pilot_preflight_passed': (
            preflight_gate.get('preflight_passed') is True
        ),
        'r8_pilot_training_authorized': (
            preflight_gate.get('r8_fold0_pilot_training_authorized') is True
        ),
        'full_fivefold_training_locked': (
            preflight_gate.get('full_fivefold_training_authorized') is False
        ),
        'pilot_version_matches': (
            preflight_gate.get('pilot_version') == R8P_PILOT_VERSION
        ),
        'pilot_code_revision_matches': (
            preflight_gate.get('pilot_code_revision')
            == R8P_PILOT_CODE_REVISION
        ),
        'r8_architecture_gate_hash_matches': (
            preflight_gate.get('r8_architecture_gate_sha256')
            == file_sha256(R8_GATE_PATH)
        ),
        'pilot_policy_hash_matches': (
            preflight_gate.get('pilot_policy_sha256')
            == file_sha256(R8P_POLICY_PATH)
        ),
        'resume_contract_hash_is_current': (
            resume_contract.get('pilot_policy_sha256')
            == file_sha256(R8P_POLICY_PATH)
        ),
        'r5_checkpoint_hash_matches': (
            preflight_gate.get('r5_checkpoint_sha256')
            == r5_checkpoint_sha256
            == r5_result.get('best_checkpoint_sha256')
        ),
        'r7_weight_migration_disabled': (
            resume_contract.get('r7_checkpoint_migration_allowed') is False
        ),
        'fold0_partition_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_and_test_counts_match': (
            len(locked_validation_df) == 23 and len(sealed_test_df) == 23
        ),
    }
    if not all(preflight_checks.values()):
        failed = [name for name, passed in preflight_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 TRAINING UPSTREAM GATE FAILED. Do not train. Failed: '
            + ', '.join(failed)
        )
    if R8P_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            'An incompatible R8 result gate already exists. Refusing to '
            'overwrite it; use a new pilot version/directory.'
        )
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 R8 fold-0 training requires CUDA.')

    R8P_PILOT_ROOT.mkdir(parents=True, exist_ok=True)
    R8T_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
    R8T_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda')
    random.seed(R8P_SEED)
    set_determinism(R8P_SEED)
    gc.collect()
    torch.cuda.empty_cache()

    r5_checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    r5_checkpoint_checks = {
        'pilot_version': (
            r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'code_revision': (
            r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
        ),
        'parameter_count': int(
            r5_checkpoint.get('parameter_count', -1)
        ) == EXPECTED_V22_PARAMETERS,
        'ema_state': 'ema_model_state' in r5_checkpoint,
    }
    if not all(r5_checkpoint_checks.values()):
        failed = [name for name, passed in r5_checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 warm start: ' + ', '.join(failed))
    r5_reference_state = {
        name: value.detach().cpu().clone()
        for name, value in r5_checkpoint['ema_model_state'].items()
    }

    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(r5_reference_state)
    if sum(parameter.numel() for parameter in model.parameters()) != (
        R8P_EXPECTED_PARAMETERS
    ):
        raise RuntimeError('R8 parameter count changed before training.')
    model.reset_satellite_gate_to_zero()
    
    
    rng_before_ema_construction = pilot_capture_rng_state()
    ema_model = VascuFateNetR8(pilot_schema).to(device).eval()
    ema_model.load_state_dict(model.state_dict(), strict=True)
    pilot_restore_rng_state(rng_before_ema_construction)
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    optimizer = torch.optim.AdamW(
        r8p_optimizer_groups(model), betas=(0.9, 0.99), eps=1e-8
    )
    dataset = Trial8R8Fold0Dataset()
    expected_updates_per_epoch = int(math.ceil(
        len(dataset) / R8P_ACCUMULATION_STEPS
    ))
    if len(dataset) != 168 or expected_updates_per_epoch != 42:
        raise RuntimeError('R8 epoch/update cardinality changed.')

    baseline = {
        'r5_mean_dice': float(r5_summary['positive_mean_dice']),
        'r5_global_dice': float(r5_summary['positive_global_dice']),
        'mean_dice_floor': float(
            r8_policy['checkpoint_eligibility']['mean_dice_floor']
        ),
        'global_dice_floor': float(
            r8_policy['checkpoint_eligibility']['global_dice_floor']
        ),
        'small_case_mean_dice': r8p_r5_small_case_dice(r5_summary),
        'small_component_recall': r8p_r5_small_component_recall(r5_baseline),
    }
    pilot_config = {
        'pilot_version': R8P_PILOT_VERSION,
        'code_revision': R8P_PILOT_CODE_REVISION,
        'implementation_revision': R8T_IMPLEMENTATION_REVISION,
        'architecture': R8_MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82,
        'heldout_subjects': 20,
        'parameter_count': R8P_EXPECTED_PARAMETERS,
        'samples_per_epoch': len(dataset),
        'optimizer_updates_per_epoch': expected_updates_per_epoch,
        'maximum_epochs': R8P_MAX_EPOCHS,
        'minimum_epochs': R8P_MINIMUM_EPOCHS,
        'early_stopping_patience': R8P_EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': (
            R8P_EARLY_STOPPING_MINIMUM_DELTA
        ),
        'early_stopping_metric': 'positive_mean_dice_checked_every_epoch',
        'accumulation_steps': R8P_ACCUMULATION_STEPS,
        'gradient_clip_norm': R8P_GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'learning_rate_policy': {
            str(epoch): r8p_learning_rates(epoch)
            for epoch in range(1, R8P_MAX_EPOCHS + 1)
        },
        'stage_policy': {
            str(epoch): r8p_stage(epoch)
            for epoch in range(1, R8P_MAX_EPOCHS + 1)
        },
        'checkpoint_baseline': baseline,
        'checkpoint_selection': (
            'eligibility_then_lexicographic_mean_small_case_'
            'small_component_lesion_f1_global'
        ),
        'warm_start': 'r5_best_ema_only',
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'r7_weights_used': False,
        'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'resume_contract_sha256': file_sha256(R8P_RESUME_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if R8P_PILOT_CONFIG_PATH.exists():
        existing = load_json(R8P_PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing R8 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, R8P_PILOT_CONFIG_PATH)
    if pilot_config_hash(load_json(R8P_PILOT_CONFIG_PATH)) != config_hash:
        raise RuntimeError('R8 pilot configuration round-trip is unstable.')

    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': R8P_PILOT_VERSION,
        'code_revision': R8P_PILOT_CODE_REVISION,
        'implementation_revision': R8T_IMPLEMENTATION_REVISION,
        'pilot_config_sha256': file_sha256(R8P_PILOT_CONFIG_PATH),
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_best_ema_used_for_initialization': True,
        'r7_weights_used': False,
        'same_r8_last_checkpoint_only_resume': True,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'upstream_checks': preflight_checks,
    }
    if R8P_PILOT_START_GATE_PATH.exists():
        existing_start = load_json(R8P_PILOT_START_GATE_PATH)
        immutable = [
            'pilot_version', 'code_revision', 'implementation_revision',
            'pilot_config_sha256', 'pilot_policy_sha256',
            'preflight_gate_sha256', 'r5_checkpoint_sha256',
        ]
        if any(existing_start.get(name) != start_gate.get(name) for name in immutable):
            raise RuntimeError(
                'Existing R8 start gate differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(start_gate, R8P_PILOT_START_GATE_PATH)

    start_epoch = 1
    optimizer_steps = 0
    best_selection_key = None
    best_epoch = 0
    best_thresholds = {'local': 0.50, 'proposal': 0.10}
    best_metrics = None
    patience_reference_mean_dice = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    resumed = False
    if R8P_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            R8P_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': (
                checkpoint.get('pilot_version') == R8P_PILOT_VERSION
            ),
            'code_revision': (
                checkpoint.get('code_revision') == R8P_PILOT_CODE_REVISION
            ),
            'implementation_revision': (
                checkpoint.get('implementation_revision')
                == R8T_IMPLEMENTATION_REVISION
            ),
            'policy_hash': (
                checkpoint.get('pilot_policy_sha256')
                == file_sha256(R8P_POLICY_PATH)
            ),
            'preflight_hash': (
                checkpoint.get('preflight_gate_sha256')
                == file_sha256(R8P_GATE_PATH)
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'r5_hash': (
                checkpoint.get('r5_checkpoint_sha256')
                == r5_checkpoint_sha256
            ),
            'parameters': int(
                checkpoint.get('parameter_count', -1)
            ) == R8P_EXPECTED_PARAMETERS,
            'r7_weights_excluded': checkpoint.get('r7_weights_used') is False,
            'required_fields': all(
                name in checkpoint
                for name in resume_contract['required_checkpoint_fields']
            ),
        }
        if not all(resume_checks.values()):
            failed = [name for name, passed in resume_checks.items() if not passed]
            raise RuntimeError('Unsafe R8 checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'], strict=True)
        ema_model.load_state_dict(checkpoint['ema_model_state'], strict=True)
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        optimizer_steps = int(checkpoint['optimizer_steps'])
        best_selection_key = checkpoint['best_selection_key']
        best_epoch = int(checkpoint.get('best_epoch', 0))
        best_thresholds = dict(checkpoint['best_thresholds'])
        best_metrics = checkpoint.get('best_metrics')
        patience_reference_mean_dice = float(
            checkpoint['patience_reference_mean_dice']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if not R8P_HISTORY_PATH.exists():
            raise RuntimeError('R8 resume checkpoint exists without history.')
        history_rows = pd.read_csv(R8P_HISTORY_PATH).to_dict('records')
        if max(int(row['epoch']) for row in history_rows) != start_epoch - 1:
            raise RuntimeError('R8 resume history/checkpoint epochs disagree.')
        if optimizer_steps != (start_epoch - 1) * expected_updates_per_epoch:
            raise RuntimeError('R8 resume optimizer-step count is inconsistent.')
        if best_epoch > 0 and not R8P_BEST_CHECKPOINT_PATH.exists():
            raise RuntimeError('R8 resume metadata names a missing best checkpoint.')
        resumed = True
    else:
        if R8P_BEST_CHECKPOINT_PATH.exists() or R8P_HISTORY_PATH.exists():
            raise RuntimeError(
                'R8 partial artifacts exist without a last checkpoint. '
                'Refusing an unsafe implicit restart.'
            )

    del r5_checkpoint
    gc.collect()
    torch.cuda.empty_cache()

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {R8P_PILOT_VERSION}')
    print(f'Implementation revision:             {R8T_IMPLEMENTATION_REVISION}')
    print('Warm start:                          R5 best EMA only')
    print('R7 weights used:                     False')
    print(f'Resume epoch:                         {start_epoch}')
    print(f'Same-run resume:                      {resumed}')
    print(f'Fit / held-out subjects:              82 / 20')
    print(f'Samples / optimizer updates per epoch:{len(dataset)} / '
          f'{expected_updates_per_epoch}')
    print(f'Parameters:                           {R8P_EXPECTED_PARAMETERS:,}')
    print('Validation and early-stop check:      every epoch')
    print(f'Minimum epochs / patience:            '
          f'{R8P_MINIMUM_EPOCHS} / {R8P_EARLY_STOPPING_PATIENCE}')
    print('Checkpoint policy:                    deployment-metric Pareto guard')
    print('Fold-0 held-out targets:              validation only')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Full five-fold training authorized:   False')
    print('R5 mean/global Dice:                  '
          f'{baseline["r5_mean_dice"]:.4f} / '
          f'{baseline["r5_global_dice"]:.4f}')
    print('Eligibility mean/global floors:       '
          f'{baseline["mean_dice_floor"]:.4f} / '
          f'{baseline["global_dice_floor"]:.4f}')

    resume_already_exhausted_patience = bool(
        resumed
        and start_epoch > R8P_MINIMUM_EPOCHS
        and validations_without_improvement
        >= R8P_EARLY_STOPPING_PATIENCE
    )
    stop_reason = (
        'early_stopping'
        if resume_already_exhausted_patience else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    training_epochs = (
        [] if resume_already_exhausted_patience
        else range(start_epoch, R8P_MAX_EPOCHS + 1)
    )
    for epoch in training_epochs:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True,
        )
        stage, learning_rates = r8p_apply_stage(model, epoch, optimizer)
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        satellite_training_rows = []
        dropout_counts = defaultdict(int)
        processed = 0
        epoch_gradient_norms = []
        epoch_optimizer_steps = 0
        for batch_index, cpu_batch in enumerate(loader):
            flags = r8t_dropout_flags(epoch, batch_index)
            for name, value in flags.items():
                dropout_counts[name] += int(bool(value))
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts, satellite_rows, output = (
                    r8p_compute_training_loss(
                        model, cpu_batch, device, epoch,
                        dropout_flags_override=flags,
                    )
                )
                scaled_loss = loss / R8P_ACCUMULATION_STEPS
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError(f'Epoch {epoch}: non-finite training loss.')
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            satellite_training_rows.extend(satellite_rows)
            del output, loss, scaled_loss, parts
            should_step = (
                (batch_index + 1) % R8P_ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    [parameter for parameter in model.parameters()
                     if parameter.requires_grad],
                    R8P_GRADIENT_CLIP_NORM,
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_steps += 1
                epoch_optimizer_steps += 1
                r8t_ema_update(ema_model, model, optimizer_steps)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))
        if epoch_optimizer_steps != expected_updates_per_epoch:
            raise RuntimeError(
                f'Epoch {epoch}: expected {expected_updates_per_epoch} '
                f'optimizer steps, observed {epoch_optimizer_steps}.'
            )

        summary, tier_rows = r8t_validate(ema_model, epoch, device)
        metrics = {
            'positive_mean_dice': float(summary['positive_mean_dice']),
            'positive_global_dice': float(summary['positive_global_dice']),
            'small_case_mean_dice': float(summary['small_case_mean_dice']),
            'small_component_recall': float(
                summary['small_component_recall']
            ),
            'lesion_f1': float(summary['lesion_f1']),
        }
        eligible, eligibility_guards, selection_key = (
            r8p_checkpoint_evaluation(metrics, baseline)
        )
        strict_best_improved = bool(
            eligible
            and (
                best_selection_key is None
                or tuple(selection_key) > tuple(best_selection_key)
            )
        )
        if strict_best_improved:
            best_selection_key = list(selection_key)
            best_epoch = int(epoch)
            best_thresholds = {
                'local': float(summary['threshold']),
                'proposal': float(summary['proposal_threshold']),
            }
            best_metrics = dict(metrics)

        current_mean_dice = float(summary['positive_mean_dice'])
        patience_improved = (
            current_mean_dice
            > patience_reference_mean_dice
            + R8P_EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_improved:
            patience_reference_mean_dice = current_mean_dice
            validations_without_improvement = 0
        elif epoch < R8P_MINIMUM_EPOCHS:
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1

        branch_frame = pd.DataFrame()
        drift = {
            'relative_l2_drift': 0.0,
            'maximum_absolute_parameter_difference': 0.0,
        }
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = r8t_branch_probe(ema_model, epoch, device)
            drift = r8t_base_drift(
                ema_model, r5_reference_state, epoch
            )

        elapsed_minutes = (time.time() - epoch_start) / 60.0
        satellite_training = {
            'train_satellite_positive_logit_mean': float(np.mean([
                row['positive_logit_mean'] for row in satellite_training_rows
            ])) if satellite_training_rows else float('nan'),
            'train_satellite_hard_negative_logit_mean': float(np.mean([
                row['hard_negative_logit_mean']
                for row in satellite_training_rows
            ])) if satellite_training_rows else float('nan'),
            'train_satellite_relative_logit_margin': float(np.mean([
                row['relative_logit_margin']
                for row in satellite_training_rows
            ])) if satellite_training_rows else float('nan'),
        }
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        history_row = {
            'epoch': int(epoch),
            'stage': stage,
            **{
                f'train_{name}': value / max(processed, 1)
                for name, value in running.items()
            },
            **satellite_training,
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'optimizer_steps_this_epoch': int(epoch_optimizer_steps),
            'optimizer_steps_total': int(optimizer_steps),
            **{
                f'learning_rate_{name}': float(value)
                for name, value in learning_rates.items()
            },
            **{
                f'dropout_count_{name}': int(value)
                for name, value in dropout_counts.items()
            },
            'validation_mean_dice': metrics['positive_mean_dice'],
            'validation_median_dice': float(
                summary['positive_median_dice']
            ),
            'validation_global_dice': metrics['positive_global_dice'],
            'validation_small_case_mean_dice': metrics[
                'small_case_mean_dice'
            ],
            'validation_lesion_precision': float(
                summary['lesion_precision']
            ),
            'validation_lesion_recall': float(summary['lesion_recall']),
            'validation_lesion_f1': metrics['lesion_f1'],
            'validation_component_recall': float(
                summary['component_recall']
            ),
            'validation_small_component_recall': metrics[
                'small_component_recall'
            ],
            'validation_satellite_margin': float(summary[
                'satellite_positive_minus_hard_negative_logit_margin'
            ]),
            'validation_satellite_gate_abs_mean': float(
                summary['satellite_gate_abs_mean']
            ),
            'validation_small_satellite_representation_failure': float(
                summary[
                    'small_satellite_component_representation_failure_fraction'
                ]
            ),
            'validation_base_to_final_target_delta': float(
                summary['base_to_final_target_mean_absolute_delta']
            ),
            **{
                f'validation_{name}_mean_dice': float(
                    bin_metrics.get(name, float('nan'))
                )
                for name in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'local_threshold': float(summary['threshold']),
            'proposal_threshold': float(summary['proposal_threshold']),
            'checkpoint_eligible': bool(eligible),
            **{
                f'eligibility_{name}': bool(value)
                for name, value in eligibility_guards.items()
            },
            'strict_eligible_best_improved': bool(strict_best_improved),
            'best_epoch': int(best_epoch),
            'patience_improved_by_minimum_delta': bool(patience_improved),
            'patience_reference_mean_dice': float(
                patience_reference_mean_dice
            ),
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'r5_base_relative_l2_drift': float(
                drift['relative_l2_drift']
            ),
            'elapsed_minutes': float(elapsed_minutes),
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != int(epoch)
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), R8P_HISTORY_PATH)
        satellite_history = pd.DataFrame([{
            'epoch': int(epoch),
            **satellite_training,
            'validation_satellite_margin': float(summary[
                'satellite_positive_minus_hard_negative_logit_margin'
            ]),
            'validation_gate_mean': float(summary['satellite_gate_mean']),
            'validation_gate_abs_mean': float(
                summary['satellite_gate_abs_mean']
            ),
            'validation_gate_minimum': float(
                summary['satellite_gate_minimum']
            ),
            'validation_gate_maximum': float(
                summary['satellite_gate_maximum']
            ),
            'validation_small_representation_failure': float(
                summary[
                    'small_satellite_component_representation_failure_fraction'
                ]
            ),
            'validation_base_to_final_target_delta': float(
                summary['base_to_final_target_mean_absolute_delta']
            ),
        }])
        if R8T_SATELLITE_HISTORY_PATH.exists():
            prior = pd.read_csv(R8T_SATELLITE_HISTORY_PATH)
            prior = prior[prior['epoch'].astype(int) != int(epoch)]
            satellite_history = pd.concat(
                [prior, satellite_history], ignore_index=True
            )
        atomic_write_csv(satellite_history, R8T_SATELLITE_HISTORY_PATH)

        checkpoint_record = {
            'pilot_version': R8P_PILOT_VERSION,
            'code_revision': R8P_PILOT_CODE_REVISION,
            'implementation_revision': R8T_IMPLEMENTATION_REVISION,
            'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
            'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
            'config_hash': config_hash,
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'r7_weights_used': False,
            'parameter_count': R8P_EXPECTED_PARAMETERS,
            'epoch': int(epoch),
            'stage': stage,
            'optimizer_steps': int(optimizer_steps),
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_selection_key': best_selection_key,
            'best_epoch': int(best_epoch),
            'best_thresholds': best_thresholds,
            'best_metrics': best_metrics,
            'patience_reference_mean_dice': float(
                patience_reference_mean_dice
            ),
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'validation_summary': summary,
            'checkpoint_eligible': bool(eligible),
            'eligibility_guards': eligibility_guards,
            'selection_key': list(selection_key),
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }
        pilot_atomic_torch_save(
            checkpoint_record, R8P_LAST_CHECKPOINT_PATH
        )
        if strict_best_improved:
            pilot_atomic_torch_save(
                checkpoint_record, R8P_BEST_CHECKPOINT_PATH
            )
        atomic_write_json({
            'pilot_version': R8P_PILOT_VERSION,
            'code_revision': R8P_PILOT_CODE_REVISION,
            'implementation_revision': R8T_IMPLEMENTATION_REVISION,
            'status': 'running',
            'last_completed_epoch': int(epoch),
            'stage': stage,
            'optimizer_steps': int(optimizer_steps),
            'best_epoch': int(best_epoch),
            'eligible_best_found': best_epoch > 0,
            'best_selection_key': best_selection_key,
            'best_thresholds': best_thresholds,
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'validation_checked_this_epoch': True,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, R8T_STATE_PATH)

        train_margin = satellite_training[
            'train_satellite_relative_logit_margin'
        ]
        print(
            f'Epoch {epoch:03d}/{R8P_MAX_EPOCHS}: stage={stage}, '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'satellite={running["satellite_objective"]/max(processed,1):.4f}, '
            f'train_margin={train_margin:.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean/median/global Dice='
            f'{summary["positive_mean_dice"]:.4f}/'
            f'{summary["positive_median_dice"]:.4f}/'
            f'{summary["positive_global_dice"]:.4f}, '
            f'lesion P/R/F1={summary["lesion_precision"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}/'
            f'{summary["lesion_f1"]:.4f}'
        )
        print(
            f'  Small cases/components: Dice='
            f'{summary["small_case_mean_dice"]:.4f}, recall='
            f'{summary["small_component_recall"]:.4f}; '
            f'all-component recall={summary["component_recall"]:.4f}'
        )
        print(
            '  Satellite: held-out margin/gate|mean|/small repr. failure='
            f'{summary["satellite_positive_minus_hard_negative_logit_margin"]:.4f}/'
            f'{summary["satellite_gate_abs_mean"]:.5f}/'
            f'{summary["small_satellite_component_representation_failure_fraction"]:.4f}, '
            f'base-to-final target delta='
            f'{summary["base_to_final_target_mean_absolute_delta"]:.5f}'
        )
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}; '
            f'eligible={eligible}, guards={eligibility_guards}, '
            f'best_epoch={best_epoch}, patience='
            f'{validations_without_improvement}/'
            f'{R8P_EARLY_STOPPING_PATIENCE}'
        )
        if len(branch_frame):
            branch_delta = branch_frame.groupby('ablation')[
                'mean_absolute_target_probability_delta'
            ].mean().to_dict()
            print(
                '  Branch target deltas: '
                + ', '.join(
                    f'{name}={float(value):.5f}'
                    for name, value in sorted(branch_delta.items())
                )
                + f'; R5-base relative drift='
                f'{drift["relative_l2_drift"]:.6f}'
            )
        last_summary, last_tier_rows = summary, tier_rows
        if (
            epoch >= R8P_MINIMUM_EPOCHS
            and validations_without_improvement
            >= R8P_EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if last_summary is None:
        if not R8P_LAST_CHECKPOINT_PATH.exists():
            raise RuntimeError('R8 training produced no checkpoint.')
        last_checkpoint = torch.load(
            R8P_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        last_summary = dict(last_checkpoint['validation_summary'])
        last_epoch = int(last_checkpoint['epoch'])
        last_tier_path = (
            R8T_VALIDATION_ROOT / f'epoch_{last_epoch:03d}'
            / 'component_tier_metrics.csv'
        )
        if not last_tier_path.exists():
            raise RuntimeError(
                'R8 last checkpoint lacks its component-tier audit.'
            )
        last_tier_rows = pd.read_csv(last_tier_path).to_dict('records')

    eligible_best_found = bool(
        best_epoch > 0 and R8P_BEST_CHECKPOINT_PATH.exists()
    )
    if eligible_best_found:
        best_checkpoint = torch.load(
            R8P_BEST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        ema_model.load_state_dict(
            best_checkpoint['ema_model_state'], strict=True
        )
        best_summary = dict(best_checkpoint['validation_summary'])
        best_epoch = int(best_checkpoint['epoch'])
        best_tier_path = (
            R8T_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
            / 'component_tier_metrics.csv'
        )
        best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
        best_branch_path = (
            R8T_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        )
        if best_branch_path.exists():
            best_branch_frame = pd.read_csv(best_branch_path)
        else:
            best_branch_frame = r8t_branch_probe(
                ema_model, best_epoch, device
            )
        best_drift = r8t_base_drift(
            ema_model, r5_reference_state, best_epoch
        )
    else:
        best_summary = dict(last_summary)
        best_tier_rows = list(last_tier_rows)
        best_branch_frame = pd.DataFrame()
        best_drift = {
            'relative_l2_drift': None,
            'maximum_absolute_parameter_difference': None,
        }

    directional_checks = {
        'eligible_r8_checkpoint_found': eligible_best_found,
        'mean_dice_not_below_r5': (
            eligible_best_found
            and float(best_summary['positive_mean_dice'])
            >= baseline['r5_mean_dice']
        ),
        'global_dice_meets_pareto_floor': (
            eligible_best_found
            and float(best_summary['positive_global_dice'])
            >= baseline['global_dice_floor']
        ),
        'small_case_dice_not_below_r5': (
            eligible_best_found
            and float(best_summary['small_case_mean_dice'])
            >= baseline['small_case_mean_dice']
        ),
        'small_component_recall_improved': (
            eligible_best_found
            and float(best_summary['small_component_recall'])
            > baseline['small_component_recall']
        ),
        'satellite_margin_is_positive': (
            eligible_best_found
            and float(best_summary[
                'satellite_positive_minus_hard_negative_logit_margin'
            ]) > 0.0
        ),
        'satellite_gate_is_active_and_bounded': (
            eligible_best_found
            and float(best_summary['satellite_gate_abs_mean']) > 1e-4
            and max(
                abs(float(best_summary['satellite_gate_minimum'])),
                abs(float(best_summary['satellite_gate_maximum'])),
            ) <= R8_SATELLITE_GATE_BOUND + 1e-5
        ),
    }
    directional_success = all(directional_checks.values())
    recommendation = r8t_recommendation(
        best_summary, baseline, eligible_best_found
    )
    branch_summary = (
        best_branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
        if len(best_branch_frame) else {}
    )
    completed_epoch = int(max(
        int(row['epoch']) for row in history_rows
    )) if history_rows else int(start_epoch - 1)
    result = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_complete': True,
        'pilot_version': R8P_PILOT_VERSION,
        'code_revision': R8P_PILOT_CODE_REVISION,
        'implementation_revision': R8T_IMPLEMENTATION_REVISION,
        'stop_reason': stop_reason,
        'last_completed_epoch': completed_epoch,
        'best_epoch': int(best_epoch) if eligible_best_found else None,
        'eligible_r8_checkpoint_found': eligible_best_found,
        'directional_success': bool(directional_success),
        'directional_checks': directional_checks,
        'best_validation_summary': best_summary,
        'best_component_tier_metrics': best_tier_rows,
        'best_branch_target_probability_deltas': branch_summary,
        'best_r5_base_parameter_drift': best_drift,
        'r5_baseline': baseline,
        'mean_dice_change_vs_r5': (
            float(best_summary['positive_mean_dice'])
            - baseline['r5_mean_dice']
        ) if eligible_best_found else None,
        'global_dice_change_vs_r5': (
            float(best_summary['positive_global_dice'])
            - baseline['r5_global_dice']
        ) if eligible_best_found else None,
        'small_case_dice_change_vs_r5': (
            float(best_summary['small_case_mean_dice'])
            - baseline['small_case_mean_dice']
        ) if eligible_best_found else None,
        'small_component_recall_change_vs_r5': (
            float(best_summary['small_component_recall'])
            - baseline['small_component_recall']
        ) if eligible_best_found else None,
        'recommendation': recommendation,
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
        'pilot_config_sha256': file_sha256(R8P_PILOT_CONFIG_PATH),
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'last_checkpoint_sha256': file_sha256(R8P_LAST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': (
            file_sha256(R8P_BEST_CHECKPOINT_PATH)
            if eligible_best_found else None
        ),
        'deployment_fallback_checkpoint': (
            str(R8P_BEST_CHECKPOINT_PATH)
            if eligible_best_found else str(R5_BEST_CHECKPOINT_PATH)
        ),
        'fold0_heldout_targets_read_for_validation': True,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
        'fold_replication_preflight_allowed': bool(directional_success),
    }
    atomic_write_json(result, R8T_FINAL_SUMMARY_PATH)
    result['final_summary_sha256'] = file_sha256(R8T_FINAL_SUMMARY_PATH)
    atomic_write_json(result, R8P_RESULT_GATE_PATH)

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Stop reason / last epoch:            {stop_reason} / {completed_epoch}')
    print(f'Eligible R8 checkpoint found:        {eligible_best_found}')
    print(f'Best eligible epoch:                 '
          f'{best_epoch if eligible_best_found else "none"}')
    print(f'Best mean/global Dice:               '
          f'{float(best_summary["positive_mean_dice"]):.4f} / '
          f'{float(best_summary["positive_global_dice"]):.4f}')
    print(f'Best small-case Dice / comp. recall: '
          f'{float(best_summary["small_case_mean_dice"]):.4f} / '
          f'{float(best_summary["small_component_recall"]):.4f}')
    print('R5 mean/global Dice:                 '
          f'{baseline["r5_mean_dice"]:.4f} / '
          f'{baseline["r5_global_dice"]:.4f}')
    print('Satellite margin / |gate| mean:      '
          f'{float(best_summary["satellite_positive_minus_hard_negative_logit_margin"]):.4f} / '
          f'{float(best_summary["satellite_gate_abs_mean"]):.5f}')
    print(f'Directional success:                 {directional_success}')
    print(f'Recommended next action:             {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                     '
          f'{R8P_BEST_CHECKPOINT_PATH if eligible_best_found else "none; retain R5"}')
    print(f'Last checkpoint:                     {R8P_LAST_CHECKPOINT_PATH}')
    print(f'History:                             {R8P_HISTORY_PATH}')
    print(f'Result gate:                         {R8P_RESULT_GATE_PATH}')


run_r8_fold0_pilot_training()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, FOLD0)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, FOLD0
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = (
    'r7_delayed_local_contrastive_volume_bounded_r1'
)
PILOT_VERSION = 'v2_2_fold0_local_contrastive_proposal_r7'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

R7_PREFLIGHT_VERSION = 'v1_delayed_small_component_local_contrastive'
R7_PREFLIGHT_CODE_REVISION = (
    'r1_precision_bounded_diffuse_shortcut_guard'
)
R7_COMPONENT_LOSS_MAX_COEFFICIENT = 0.10
R7_COMPONENT_LOSS_START_EPOCH = 8
R7_COMPONENT_LOSS_FULL_EPOCH = 12
R7_POSITIVE_WEIGHT = 0.40
R7_HARD_NEGATIVE_WEIGHT = 0.40
R7_RANK_WEIGHT = 0.20
R7_INNER_RADIUS_MM = 4.0
R7_OUTER_RADIUS_MM = 12.0
R7_HARD_NEGATIVE_RATIO = 4
R7_HARD_NEGATIVE_MINIMUM = 16
R7_LOGIT_MARGIN = 1.0
R7_VOLUME_BUDGET_MULTIPLIER = 3.0
R7_SMALL_COMPONENT_TIERS = set(COMPONENT_TIERS[:4])
R5_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
R5_PILOT_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'
R7_GLOBAL_SPACING_MM = tuple(
    native * factor
    for native, factor in zip((0.8, 0.8, 2.0), GLOBAL_DOWNSAMPLE)
)

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'
R7_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_2_global_local_arterial'
    / 'r7_local_contrastive_component_proposal' / R7_PREFLIGHT_VERSION
)
R7_PREFLIGHT_GATE_PATH = (
    R7_PREFLIGHT_ROOT / 'r7_local_contrastive_preflight_gate.json'
)
R7_POLICY_PATH = R7_PREFLIGHT_ROOT / 'r7_local_contrastive_policy.json'
R5_PILOT_ROOT = PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / R5_PILOT_VERSION
R5_RESULT_GATE_PATH = R5_PILOT_ROOT / 'pilot_result_gate.json'
R5_BEST_CHECKPOINT_PATH = R5_PILOT_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / PILOT_VERSION
)
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'pilot_result_gate.json'
PILOT_TARGET_AUDIT_PATH = PILOT_ROOT / 'fold0_target_access_audit.csv'
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'
R7_R5_BASELINE_DIAGNOSTIC_PATH = (
    PILOT_DIAGNOSTIC_ROOT / 'r5_volume_bounded_baseline.json'
)
R7_FAILURE_DIAGNOSTIC_VERSION = 'v1_r5_r7_failure_mechanism_audit'
R7_FAILURE_DIAGNOSTIC_CODE_REVISION = (
    'r1_exposure_resolution_margin_volume_branch_checkpoint'
)
R7_FAILURE_DIAGNOSTIC_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'model_diagnostics'
    / MODEL_VERSION / R7_FAILURE_DIAGNOSTIC_VERSION
)
R7_FAILURE_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
R7_FAILURE_GATE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_diagnostic_gate.json'
)
R7_FAILURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r7_failure_summary.json'
)



PILOT_TARGET_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'fold0_target_access_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_schedule_audit.csv'
)
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    BLOCK1_COMPONENT_AUDIT_PATH, MULTIMODAL_GATE_PATH,
    MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
    FIVEFOLD_GATE_PATH, R7_PREFLIGHT_GATE_PATH, R7_POLICY_PATH,
    R5_RESULT_GATE_PATH, R5_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 fold-0 pilot prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
r7_preflight_gate = load_json(R7_PREFLIGHT_GATE_PATH)
r7_policy = load_json(R7_POLICY_PATH)
r5_result_gate = load_json(R5_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r7_component_preflight_passed': (
        r7_preflight_gate.get('preflight_passed') is True
    ),
    'r7_component_preflight_all_checks_true': (
        len(r7_preflight_gate.get('checks', {})) > 0
        and all(bool(value) for value in r7_preflight_gate['checks'].values())
    ),
    'r7_preflight_version_matches': (
        r7_preflight_gate.get('preflight_version') == R7_PREFLIGHT_VERSION
    ),
    'r7_preflight_revision_matches': (
        r7_preflight_gate.get('code_revision')
        == R7_PREFLIGHT_CODE_REVISION
    ),
    'r7_implementation_allowed': (
        r7_preflight_gate.get('r7_pilot_implementation_allowed') is True
    ),
    'r7_training_was_locked_before_this_block': (
        r7_preflight_gate.get('r7_pilot_training_authorized') is False
    ),
    'r7_architecture_parameter_count_unchanged': (
        int(r7_preflight_gate.get('architecture_parameters', -1))
        == EXPECTED_V22_PARAMETERS
        and int(r7_preflight_gate.get('new_trainable_parameters', -1)) == 0
    ),
    'r7_preflight_target_scope_is_fit_only': (
        r7_preflight_gate.get('target_arrays_read_splits') == ['fold0_fit']
        and r7_preflight_gate.get('fold0_heldout_targets_read') is False
        and r7_preflight_gate.get('locked_validation_targets_read') is False
        and r7_preflight_gate.get('test_targets_read') is False
    ),
    'r7_policy_path_matches_gate': (
        str(r7_preflight_gate.get('policy_path')) == str(R7_POLICY_PATH)
    ),
    'r7_policy_local_contrastive_contract_matches': (
        r7_policy.get('selected_component_objective', {}).get(
            'eligible_tiers'
        ) == sorted(R7_SMALL_COMPONENT_TIERS)
        and int(r7_policy.get('selected_component_objective', {}).get(
            'start_epoch', -1
        )) == R7_COMPONENT_LOSS_START_EPOCH
        and int(r7_policy.get('selected_component_objective', {}).get(
            'full_coefficient_epoch', -1
        )) == R7_COMPONENT_LOSS_FULL_EPOCH
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('maximum_coefficient', float('nan'))),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('inner_radius_mm', float('nan'))),
            R7_INNER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('outer_radius_mm', float('nan'))),
            R7_OUTER_RADIUS_MM, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('positive_weight', float('nan'))),
            R7_POSITIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_weight', float('nan'))),
            R7_HARD_NEGATIVE_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('rank_weight', float('nan'))),
            R7_RANK_WEIGHT, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_ratio', -1)) == R7_HARD_NEGATIVE_RATIO
        and int(r7_policy.get(
            'selected_component_objective', {}
        ).get('hard_negative_minimum', -1)) == R7_HARD_NEGATIVE_MINIMUM
        and math.isclose(float(r7_policy.get(
            'selected_component_objective', {}
        ).get('logit_margin', float('nan'))),
            R7_LOGIT_MARGIN, rel_tol=0.0, abs_tol=1e-12)
        and int(r7_policy.get('experiment_seed', -1)) == PILOT_SEED
    ),
    'r7_policy_requires_volume_bounded_diagnostics': (
        r7_policy.get('proposal_diagnostics', {}).get(
            'raw_distance_metrics_are_not_success_gates'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_proposal_precision'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_predicted_to_target_volume_ratio'
        ) is True
        and r7_policy.get('proposal_diagnostics', {}).get(
            'report_voxel_average_precision'
        ) is True
        and math.isclose(float(r7_policy.get(
            'proposal_diagnostics', {}
        ).get('volume_budget_multiplier', float('nan'))),
            R7_VOLUME_BUDGET_MULTIPLIER, rel_tol=0.0, abs_tol=1e-12)
    ),
    'r7_policy_preserves_whole_mask_loss': (
        math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('soft_dice_weight', float('nan'))),
            0.65, rel_tol=0.0, abs_tol=1e-12,
        )
        and math.isclose(
            float(r7_policy.get(
                'whole_mask_proposal_objective_preserved', {}
            ).get('focal_bce_weight', float('nan'))),
            0.35, rel_tol=0.0, abs_tol=1e-12,
        )
    ),
    'r7_checkpoint_policy_is_decoupled': (
        r7_policy.get('checkpoint_policy', {}).get('save_checkpoint_on')
        == 'every strict numeric score improvement'
        and r7_policy.get('checkpoint_policy', {}).get('minimum_delta_use')
        == 'early-stopping patience reset only'
        and r7_policy.get('checkpoint_policy', {}).get(
            'minimum_delta_must_not_block_checkpoint_save'
        ) is True
    ),
    'r5_read_only_baseline_gate_matches': (
        r5_result_gate.get('pilot_version') == R5_PILOT_VERSION
        and r5_result_gate.get('code_revision') == R5_PILOT_CODE_REVISION
        and r5_result_gate.get('pilot_complete') is True
        and str(r5_result_gate.get('best_checkpoint_path'))
        == str(R5_BEST_CHECKPOINT_PATH)
        and r5_result_gate.get('best_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
fold0_fit_df = development_df[
    development_df['development_fold'] != FOLD0
].copy().sort_values('subject_id')
fold0_heldout_df = development_df[
    development_df['development_fold'] == FOLD0
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
if (len(fold0_fit_df), len(fold0_heldout_df)) != (82, 20):
    raise RuntimeError('Fold-0 must contain exactly 82 fit and 20 held-out cases.')
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in fold0_fit_df.to_dict('records')
}
heldout_rows = fold0_heldout_df.to_dict('records')






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in fold0_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': 'fold0_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)

component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Block-1 component audit contains non-fit subjects.')
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
if len(positive_subjects) != 81:
    raise RuntimeError(
        f'Expected 81 positive fit subjects, observed {len(positive_subjects)}.'
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete atlas-enabled fold-0 fit pool supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            'The complete fold-0 fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def r7_physical_ball(spacing_mm, radius_mm):
    spacing = np.asarray(spacing_mm, dtype=np.float64)
    radii = np.ceil(float(radius_mm) / spacing).astype(np.int64)
    coordinates = np.meshgrid(*[
        np.arange(-radius, radius + 1, dtype=np.float64)
        for radius in radii
    ], indexing='ij')
    squared_distance = np.zeros_like(coordinates[0], dtype=np.float64)
    for axis, coordinate in enumerate(coordinates):
        squared_distance += (coordinate * spacing[axis]) ** 2
    return squared_distance <= float(radius_mm) ** 2 + 1e-8


R7_INNER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_INNER_RADIUS_MM
)
R7_OUTER_STRUCTURE = r7_physical_ball(
    R7_GLOBAL_SPACING_MM, R7_OUTER_RADIUS_MM
)


def r7_component_coefficient(epoch):
    epoch = int(epoch)
    if epoch < R7_COMPONENT_LOSS_START_EPOCH:
        return 0.0
    progress = (
        (epoch - R7_COMPONENT_LOSS_START_EPOCH + 1)
        / (R7_COMPONENT_LOSS_FULL_EPOCH - R7_COMPONENT_LOSS_START_EPOCH + 1)
    )
    return float(
        R7_COMPONENT_LOSS_MAX_COEFFICIENT * np.clip(progress, 0.0, 1.0)
    )


R7_COMPONENT_COEFFICIENT_SCHEDULE = {
    epoch: r7_component_coefficient(epoch)
    for epoch in range(1, MAX_EPOCHS + 1)
}
R7_EXPECTED_RAMP = {
    8: 0.02, 9: 0.04, 10: 0.06, 11: 0.08, 12: 0.10,
}
if not all(
    R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch] == 0.0
    for epoch in range(1, R7_COMPONENT_LOSS_START_EPOCH)
):
    raise RuntimeError('R7 local-contrastive coefficient is nonzero before epoch 8.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch], expected,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch, expected in R7_EXPECTED_RAMP.items()
):
    raise RuntimeError('R7 local-contrastive coefficient ramp changed.')
if not all(
    math.isclose(
        R7_COMPONENT_COEFFICIENT_SCHEDULE[epoch],
        R7_COMPONENT_LOSS_MAX_COEFFICIENT,
        rel_tol=0.0, abs_tol=1e-12,
    )
    for epoch in range(R7_COMPONENT_LOSS_FULL_EPOCH, MAX_EPOCHS + 1)
):
    raise RuntimeError('R7 local-contrastive coefficient does not remain capped.')


def r7_component_regions(exact_target, full_target, valid_domain):
    exact = np.asarray(exact_target, dtype=bool)
    full = np.asarray(full_target) > 0
    domain = np.asarray(valid_domain, dtype=bool)
    inner = ndimage.binary_dilation(exact, structure=R7_INNER_STRUCTURE)
    outer = ndimage.binary_dilation(exact, structure=R7_OUTER_STRUCTURE)
    negative = outer & ~inner & domain & ~full
    return inner, outer, negative


def pilot_extract_targets(
    row, local_origin, sample_type, atomic_component_id, component_tier
):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary > 0
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode {supervision_mode!r}.'
        )
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(binary, dtype=np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if start_a != start_b or start_a != start_c:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_domain = F.max_pool3d(
        torch.from_numpy(domain_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0

    selected_sample = str(sample_type) == 'positive'
    selected_valid = (
        selected_sample and str(component_tier) in R7_SMALL_COMPONENT_TIERS
    )
    selected_component = np.zeros_like(component_support, dtype=bool)
    if selected_sample:
        labels, count = ndimage.label(
            component_support, structure=np.ones((3, 3, 3), dtype=bool)
        )
        expected_count = len(component_records.get(subject_id, []))
        if count != expected_count:
            raise RuntimeError(
                f'{subject_id}: reconstructed {count} components but the '
                f'frozen audit contains {expected_count}.'
            )
        component_id = int(atomic_component_id)
        if component_id < 1 or component_id > count:
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is outside '
                f'the frozen 1..{count} range.'
            )
        selected_component = labels == component_id
        if not bool(selected_component.any()):
            raise RuntimeError(
                f'{subject_id}: selected component {component_id} is empty.'
            )

    selected_canvas, selected_start = symmetric_pad(
        selected_component.astype(np.float32),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if selected_start != start_a:
        raise RuntimeError(
            f'{subject_id}: selected-component padding start disagrees.'
        )
    selected_global = F.max_pool3d(
        torch.from_numpy(selected_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy() > 0
    selected_negative = np.zeros_like(selected_global, dtype=bool)
    if selected_valid:
        _, _, selected_negative = r7_component_regions(
            selected_global, global_target[0], global_domain
        )
        if not bool(selected_negative.any()):
            raise RuntimeError(
                f'{subject_id}: selected component has no lesion-free '
                '4-12 mm negative annulus.'
            )
    if selected_sample:
        frozen = [
            item for item in component_records[subject_id]
            if int(item['atomic_component_id']) == int(atomic_component_id)
        ]
        if len(frozen) != 1:
            raise RuntimeError(
                f'{subject_id}: selected component does not map uniquely to '
                'the frozen component audit.'
            )
        expected_global_voxels = int(frozen[0]['global_proposal_voxels'])
        if int(selected_global.sum()) != expected_global_voxels:
            raise RuntimeError(
                f'{subject_id}: selected component has '
                f'{int(selected_global.sum())} pooled voxels; expected '
                f'{expected_global_voxels}.'
            )
        if bool((selected_negative & selected_global).any()):
            raise RuntimeError(
                f'{subject_id}: negative annulus overlaps exact component.'
            )
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'selected_component_target': selected_global[None].astype(bool),
        'selected_component_negative_annulus': (
            selected_negative[None].astype(bool)
        ),
        'selected_component_valid': np.asarray(
            selected_valid, dtype=bool
        ),
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'selected_component_target': targets[
                'selected_component_target'
            ],
            'selected_component_negative_annulus': targets[
                'selected_component_negative_annulus'
            ],
            'selected_component_valid': targets[
                'selected_component_valid'
            ],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def r7_local_contrastive_loss(
    logits, exact_component_target, negative_annulus_target, component_valid,
):
    exact_target = exact_component_target.bool()
    negative_target = negative_annulus_target.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    positive_losses = []
    negative_losses = []
    rank_losses = []
    positive_logits = []
    hard_negative_logits = []
    selected_negative_counts = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]].float()
        negative = logits[batch_index][negative_target[batch_index]].float()
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError(
                'A valid r7 component must have positive voxels and a '
                'lesion-free negative annulus.'
            )
        hard_count = min(
            int(negative.numel()),
            max(
                R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * int(positive.numel()),
            ),
        )
        hard_negative = torch.topk(
            negative, k=hard_count, largest=True
        ).values
        positive_loss = F.softplus(-positive).mean()
        negative_loss = F.softplus(hard_negative).mean()
        rank_loss = F.softplus(
            R7_LOGIT_MARGIN - positive.mean() + hard_negative.mean()
        )
        contrastive_loss = (
            R7_POSITIVE_WEIGHT * positive_loss
            + R7_HARD_NEGATIVE_WEIGHT * negative_loss
            + R7_RANK_WEIGHT * rank_loss
        )
        losses.append(contrastive_loss)
        positive_losses.append(positive_loss)
        negative_losses.append(negative_loss)
        rank_losses.append(rank_loss)
        positive_logits.append(positive.mean())
        hard_negative_logits.append(hard_negative.mean())
        selected_negative_counts.append(hard_count)

    if not losses:
        zero = logits.float().sum() * 0.0
        return zero, {
            'local_contrastive_loss': zero,
            'component_positive_loss': zero,
            'component_hard_negative_loss': zero,
            'component_rank_loss': zero,
            'mean_component_positive_logit': zero.detach(),
            'mean_component_hard_negative_logit': zero.detach(),
            'selected_hard_negative_voxels': zero.detach(),
            'selected_components_in_batch': zero.detach(),
        }
    return torch.stack(losses).mean(), {
        'local_contrastive_loss': torch.stack(losses).mean(),
        'component_positive_loss': torch.stack(positive_losses).mean(),
        'component_hard_negative_loss': torch.stack(negative_losses).mean(),
        'component_rank_loss': torch.stack(rank_losses).mean(),
        'mean_component_positive_logit': torch.stack(positive_logits).mean(),
        'mean_component_hard_negative_logit': torch.stack(
            hard_negative_logits
        ).mean(),
        'selected_hard_negative_voxels': torch.as_tensor(
            float(sum(selected_negative_counts)), device=logits.device
        ),
        'selected_components_in_batch': torch.as_tensor(
            float(len(losses)), device=logits.device
        ),
    }


def r7_local_contrastive_proposal_loss(
    logits, full_target, exact_component_target,
    negative_annulus_target, component_valid, epoch,
):
    whole_dice = pilot_soft_dice_loss(logits, full_target).mean()
    whole_bce = pilot_focal_bce_loss(logits, full_target).mean()
    whole_loss = 0.65 * whole_dice + 0.35 * whole_bce
    contrastive_loss, contrastive_parts = r7_local_contrastive_loss(
        logits, exact_component_target, negative_annulus_target,
        component_valid,
    )
    coefficient = r7_component_coefficient(epoch)
    proposal_loss = whole_loss + coefficient * contrastive_loss
    return proposal_loss, {
        'proposal_whole_mask_loss': whole_loss,
        'proposal_dice_loss': whole_dice,
        'proposal_focal_bce_loss': whole_bce,
        **contrastive_parts,
        'component_loss_coefficient': torch.as_tensor(
            coefficient, device=logits.device, dtype=torch.float32
        ),
    }


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(
    model, cpu_batch, device, epoch, batch_index,
    dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    if dropout_flags_override is not None:
        required_flags = {
            'force_atlas_dropout', 'force_vascular_dropout',
            'force_occlusion_dropout', 'force_clinical_dropout',
            'force_global_dropout', 'force_arterial_dropout',
        }
        if set(dropout_flags_override) != required_flags:
            raise RuntimeError(
                'Dropout override keys do not match the frozen branch set.'
            )
        flags = {
            name: bool(dropout_flags_override[name])
            for name in sorted(required_flags)
        }
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_loss, proposal_parts = r7_local_contrastive_proposal_loss(
        output['global_proposal_logits'], global_target,
        batch['selected_component_target'],
        batch['selected_component_negative_annulus'],
        batch['selected_component_valid'], epoch,
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        **proposal_parts,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'arterial_valid_nodes': arterial_valid.sum().detach().float(),
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'arterial_dropped': torch.as_tensor(float(flags['force_arterial_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_volume_bounded_prediction(probability, target_voxels):
    probability = np.asarray(probability, dtype=np.float32)
    budget = min(
        probability.size,
        max(1, int(math.ceil(
            R7_VOLUME_BUDGET_MULTIPLIER * int(target_voxels)
        ))),
    )
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape), budget


def pilot_histogram_average_precision(positive_histogram, negative_histogram):
    positive = np.asarray(positive_histogram, dtype=np.float64)[::-1]
    negative = np.asarray(negative_histogram, dtype=np.float64)[::-1]
    total_positive = float(positive.sum())
    if total_positive <= 0:
        return float('nan')
    cumulative_positive = np.cumsum(positive)
    cumulative_total = np.cumsum(positive + negative)
    precision = cumulative_positive / np.maximum(cumulative_total, 1.0)
    return float((precision * positive).sum() / total_positive)


def pilot_threshold_metrics(case_cache, threshold, probability_key):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    voxel_precision = total_intersection / max(total_prediction, 1)
    predicted_to_target_volume_ratio = (
        total_prediction / max(total_target, 1)
    )
    return {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'voxel_precision': float(voxel_precision),
        'predicted_to_target_volume_ratio': float(
            predicted_to_target_volume_ratio
        ),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-0 held-out inference {index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'probability')
        for threshold in LOCAL_THRESHOLDS
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    proposal_threshold_df = pd.DataFrame([
        pilot_threshold_metrics(case_cache, threshold, 'proposal_probability')
        for threshold in PROPOSAL_THRESHOLDS
    ])
    proposal_threshold_df['volume_calibration_score'] = np.exp(-np.abs(
        np.log(np.maximum(
            proposal_threshold_df['predicted_to_target_volume_ratio'], 1e-6
        ))
    ))
    proposal_threshold_df['proposal_selection_score'] = (
        0.35 * proposal_threshold_df['positive_global_dice']
        + 0.15 * proposal_threshold_df['lesion_f1']
        + 0.20 * proposal_threshold_df['voxel_precision']
        + 0.15 * proposal_threshold_df['component_recall']
        + 0.15 * proposal_threshold_df['volume_calibration_score']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        [
            'proposal_selection_score', 'positive_global_dice',
            'voxel_precision', 'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    proposal_ap_positive_histogram = np.zeros(256, dtype=np.int64)
    proposal_ap_negative_histogram = np.zeros(256, dtype=np.int64)
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        proposal_intersection = int((proposal_prediction & target).sum())
        proposal_predicted_voxels = int(proposal_prediction.sum())
        proposal_distance = None
        if bool(proposal_prediction.any()):
            proposal_distance = ndimage.distance_transform_edt(
                ~proposal_prediction, sampling=case['spacing']
            )
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        budgeted_proposal_prediction = None
        budgeted_proposal_distance = None
        proposal_volume_budget_voxels = 0
        if hard_eligible and target_voxels > 0:
            (
                budgeted_proposal_prediction,
                proposal_volume_budget_voxels,
            ) = pilot_volume_bounded_prediction(proposal, target_voxels)
            budgeted_proposal_distance = ndimage.distance_transform_edt(
                ~budgeted_proposal_prediction, sampling=case['spacing']
            )
            proposal_ap_positive_histogram += np.histogram(
                proposal[target], bins=256, range=(0.0, 1.0)
            )[0]
            proposal_ap_negative_histogram += np.histogram(
                proposal[~target], bins=256, range=(0.0, 1.0)
            )[0]
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
            'proposal_predicted_voxels': proposal_predicted_voxels,
            'proposal_intersection_voxels': proposal_intersection,
            'proposal_voxel_precision': (
                proposal_intersection / max(proposal_predicted_voxels, 1)
            ),
            'proposal_voxel_recall': (
                proposal_intersection / max(target_voxels, 1)
            ),
            'proposal_predicted_to_target_volume_ratio': (
                proposal_predicted_voxels / max(target_voxels, 1)
            ),
            'proposal_volume_budget_voxels': proposal_volume_budget_voxels,
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            budgeted_proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    budgeted_proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                budgeted_proposal_max_iou = float(
                    budgeted_proposal_iou[component_id][
                        'maximum_component_iou'
                    ]
                )
                proposal_distance_mm = float('nan')
                if proposal_distance is not None:
                    proposal_distance_mm = float(
                        proposal_distance[component_slice][component_local].min(
                            initial=np.inf
                        )
                    )
                budgeted_proposal_distance_mm = float(
                    budgeted_proposal_distance[component_slice][
                        component_local
                    ].min(initial=np.inf)
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                budgeted_proposal_covered = (
                    budgeted_proposal_max_iou >= 0.05
                )
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += int(budgeted_proposal_covered)
                tier_counts[tier][3] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'proposal_minimum_distance_mm': proposal_distance_mm,
                    'budgeted_proposal_maximum_component_iou': (
                        budgeted_proposal_max_iou
                    ),
                    'budgeted_proposal_minimum_distance_mm': (
                        budgeted_proposal_distance_mm
                    ),
                    'proposal_within_2mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 2.0
                    ),
                    'proposal_within_4mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 4.0
                    ),
                    'proposal_within_8mm': bool(
                        np.isfinite(proposal_distance_mm)
                        and proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_within_2mm': bool(
                        budgeted_proposal_distance_mm <= 2.0
                    ),
                    'budgeted_proposal_within_4mm': bool(
                        budgeted_proposal_distance_mm <= 4.0
                    ),
                    'budgeted_proposal_within_8mm': bool(
                        budgeted_proposal_distance_mm <= 8.0
                    ),
                    'budgeted_proposal_covered': bool(
                        budgeted_proposal_covered
                    ),
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, budgeted_proposal_tp, total = tier_counts[tier]
        tier_components = [
            item for item in component_rows
            if item['component_tier'] == tier
        ]
        finite_distances = [
            float(item['proposal_minimum_distance_mm'])
            for item in tier_components
            if np.isfinite(item['proposal_minimum_distance_mm'])
        ]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
            'budgeted_proposal_component_recall': (
                budgeted_proposal_tp / max(total, 1)
            ),
            'proposal_within_2mm_fraction': float(np.mean([
                item['proposal_within_2mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_4mm_fraction': float(np.mean([
                item['proposal_within_4mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'proposal_within_8mm_fraction': float(np.mean([
                item['proposal_within_8mm'] for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_2mm_fraction': float(np.mean([
                item['budgeted_proposal_within_2mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_4mm_fraction': float(np.mean([
                item['budgeted_proposal_within_4mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'budgeted_proposal_within_8mm_fraction': float(np.mean([
                item['budgeted_proposal_within_8mm']
                for item in tier_components
            ])) if tier_components else 0.0,
            'median_finite_proposal_distance_mm': float(
                np.median(finite_distances)
            ) if finite_distances else float('nan'),
            'mean_proposal_maximum_probability': float(np.mean([
                item['proposal_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'mean_final_maximum_probability': float(np.mean([
                item['final_probability_maximum']
                for item in tier_components
            ])) if tier_components else 0.0,
            'representation_failure_fraction': float(np.mean([
                item['representation_failure_below_0p10']
                for item in tier_components
            ])) if tier_components else 0.0,
        })
    small_component_rows = [
        item for item in component_rows
        if item['component_tier'] in set(COMPONENT_TIERS[:4])
    ]
    proposal_voxel_average_precision = pilot_histogram_average_precision(
        proposal_ap_positive_histogram, proposal_ap_negative_histogram
    )
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'proposal_voxel_precision': float(
            proposal_selected['voxel_precision']
        ),
        'proposal_predicted_to_target_volume_ratio': float(
            proposal_selected['predicted_to_target_volume_ratio']
        ),
        'proposal_volume_calibration_score': float(
            proposal_selected['volume_calibration_score']
        ),
        'proposal_voxel_average_precision_256bin': float(
            proposal_voxel_average_precision
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'small_components_audited': len(small_component_rows),
        'small_component_proposal_within_2mm_fraction': float(np.mean([
            item['proposal_within_2mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_4mm_fraction': float(np.mean([
            item['proposal_within_4mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_proposal_within_8mm_fraction': float(np.mean([
            item['proposal_within_8mm'] for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_budgeted_proposal_within_2mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_2mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_4mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_4mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'small_component_budgeted_proposal_within_8mm_fraction': float(
            np.mean([
                item['budgeted_proposal_within_8mm']
                for item in small_component_rows
            ]) if small_component_rows else 0.0
        ),
        'budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered'] for item in component_rows
        ])) if component_rows else 0.0,
        'small_budgeted_proposal_component_recall': float(np.mean([
            item['budgeted_proposal_covered']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'small_component_representation_failure_fraction': float(np.mean([
            item['representation_failure_below_0p10']
            for item in small_component_rows
        ])) if small_component_rows else 0.0,
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    required_finite_summary_metrics = [
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'proposal_global_dice', 'proposal_component_recall',
        'proposal_voxel_precision',
        'proposal_predicted_to_target_volume_ratio',
        'proposal_voxel_average_precision_256bin',
        'budgeted_proposal_component_recall',
        'small_budgeted_proposal_component_recall',
        'small_component_budgeted_proposal_within_2mm_fraction',
        'small_component_budgeted_proposal_within_4mm_fraction',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'small_component_representation_failure_fraction',
        'pilot_checkpoint_score',
    ]
    nonfinite_summary_metrics = [
        name for name in required_finite_summary_metrics
        if not np.isfinite(float(summary[name]))
    ]
    if nonfinite_summary_metrics:
        raise RuntimeError(
            'Validation produced non-finite required metrics: '
            + ', '.join(nonfinite_summary_metrics)
        )
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = fold0_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_fold0_pilot():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('pilot_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
            and completed.get('best_checkpoint_sha256')
            == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 R7 FOLD-0 PILOT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No checkpoint or diagnostic output was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 pilot requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22Fold0Dataset()
    smoke_index = None
    smoke_subject_id = None
    for candidate_index, entry in enumerate(dataset.schedule):
        if entry['sample_type'] != 'positive':
            continue
        candidate_subject = str(entry['subject_id'])
        if not as_bool(
            fit_row_by_subject[candidate_subject]['trial8_atlas_enabled']
        ):
            continue
        candidate_sample = dataset[candidate_index]
        candidate_checks = {
            'atlas_enabled': bool(np.asarray(
                candidate_sample['atlas_enabled']
            ).reshape(-1)[0]),
            'territory_nodes_valid': bool(np.asarray(
                candidate_sample['territory_valid']
            ).astype(bool).any()),
            'vessel_nodes_valid': bool(np.asarray(
                candidate_sample['vessel_valid']
            ).astype(bool).any()),
            'selected_component_valid': bool(np.asarray(
                candidate_sample['selected_component_valid']
            ).reshape(-1)[0]),
        }
        if all(candidate_checks.values()):
            smoke_index = int(candidate_index)
            smoke_subject_id = candidate_subject
            break
    if smoke_index is None:
        raise RuntimeError(
            'No positive fold-0-fit sample has simultaneous atlas, territory, '
            'vessel, and selected-component support for the all-branch smoke '
            'test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    smoke_dropout_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device,
            epoch=R7_COMPONENT_LOSS_FULL_EPOCH, batch_index=0,
            dropout_flags_override=smoke_dropout_flags,
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 pilot smoke loss is non-finite.')
    r7_smoke_checks = {
        'positive_sample_selected': bool(
            smoke_batch['selected_component_valid'].bool().all().item()
        ),
        'atlas_enabled_for_smoke': bool(
            smoke_batch['atlas_enabled'].bool().all().item()
        ),
        'territory_support_for_smoke': bool(
            smoke_batch['territory_valid'].bool().any().item()
        ),
        'vessel_support_for_smoke': bool(
            smoke_batch['vessel_valid'].bool().any().item()
        ),
        'projected_arterial_nodes_valid_for_smoke': float(
            smoke_parts['arterial_valid_nodes'].detach().float().cpu()
        ) > 0.0,
        'arterial_loss_finite_for_smoke': bool(torch.isfinite(
            smoke_parts['arterial_loss']
        ).item()),
        'all_branch_dropouts_disabled_for_smoke': not any(
            smoke_dropout_flags.values()
        ),
        'exact_component_target_nonempty': bool(
            smoke_batch['selected_component_target'].bool().any().item()
        ),
        'negative_annulus_nonempty': bool(
            smoke_batch['selected_component_negative_annulus']
            .bool().any().item()
        ),
        'negative_annulus_disjoint_from_exact_component': not bool(torch.any(
            smoke_batch['selected_component_target'].bool()
            & smoke_batch['selected_component_negative_annulus'].bool()
        ).item()),
        'local_contrastive_loss_positive': float(
            smoke_parts['local_contrastive_loss']
            .detach().float().cpu()
        ) > 0.0,
        'smoke_uses_maximum_contrastive_coefficient': math.isclose(
            float(smoke_parts['component_loss_coefficient']
                  .detach().float().cpu()),
            R7_COMPONENT_LOSS_MAX_COEFFICIENT,
            rel_tol=0.0, abs_tol=1e-7,
        ),
        'whole_mask_proposal_loss_finite': bool(torch.isfinite(
            smoke_parts['proposal_whole_mask_loss']
        ).item()),
    }
    if not all(r7_smoke_checks.values()):
        failed = [
            name for name, passed in r7_smoke_checks.items() if not passed
        ]
        raise RuntimeError(
            'R7 local-contrastive smoke checks failed: ' + ', '.join(failed)
        )
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    if R7_R5_BASELINE_DIAGNOSTIC_PATH.exists():
        r5_baseline_diagnostic = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
        if (
            r5_baseline_diagnostic.get('r5_checkpoint_sha256')
            != r5_checkpoint_sha256
            or r5_baseline_diagnostic.get('metric_contract_revision')
            != PILOT_CODE_REVISION
        ):
            raise RuntimeError(
                'Existing r5 volume-bounded baseline is incompatible with '
                'the current r7 contract.'
            )
        r5_recomputed_summary = dict(
            r5_baseline_diagnostic['validation_summary']
        )
    else:
        rng_state_before_baseline = pilot_capture_rng_state()
        r5_checkpoint = torch.load(
            R5_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
        )
        r5_checkpoint_checks = {
            'pilot_version': (
                r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
            ),
            'code_revision': (
                r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
            ),
            'parameter_count': int(
                r5_checkpoint.get('parameter_count', -1)
            ) == EXPECTED_V22_PARAMETERS,
        }
        if not all(r5_checkpoint_checks.values()):
            failed = [
                name for name, passed in r5_checkpoint_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Unsafe r5 read-only baseline checkpoint: '
                + ', '.join(failed)
            )
        r5_baseline_model = VascuFateNetV22(pilot_schema).to(device).eval()
        r5_baseline_model.load_state_dict(r5_checkpoint['ema_model_state'])
        for parameter in r5_baseline_model.parameters():
            parameter.requires_grad_(False)
        print(
            '\nRecomputing the completed r5 best checkpoint with the r7 '
            'precision- and volume-bounded diagnostic contract.'
        )
        r5_recomputed_summary, r5_recomputed_tiers = pilot_validate(
            r5_baseline_model, epoch=0, device=device
        )
        r5_baseline_diagnostic = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'metric_contract_revision': PILOT_CODE_REVISION,
            'r5_pilot_version': R5_PILOT_VERSION,
            'r5_code_revision': R5_PILOT_CODE_REVISION,
            'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'validation_summary': r5_recomputed_summary,
            'component_tier_metrics': r5_recomputed_tiers,
            'r5_checkpoint_loaded_into_r7_model': False,
            'optimizer_steps': 0,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
        }
        atomic_write_json(
            r5_baseline_diagnostic, R7_R5_BASELINE_DIAGNOSTIC_PATH
        )
        del r5_baseline_model, r5_checkpoint
        gc.collect()
        torch.cuda.empty_cache()
        pilot_restore_rng_state(rng_state_before_baseline)

    
    
    
    
    del model, ema_model
    gc.collect()
    torch.cuda.empty_cache()
    set_determinism(PILOT_SEED)
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed during lineage reset.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    lineage_smoke_loader = torch.utils.data.DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True
    )
    lineage_smoke_batch = next(iter(lineage_smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        lineage_smoke_loss, lineage_smoke_parts = (
            pilot_compute_training_loss(
                model, lineage_smoke_batch, device, epoch=1, batch_index=0
            )
        )
    if not bool(torch.isfinite(lineage_smoke_loss).item()):
        raise RuntimeError('R7 controlled-lineage smoke loss is non-finite.')
    if float(
        lineage_smoke_parts['component_loss_coefficient']
        .detach().float().cpu()
    ) != 0.0:
        raise RuntimeError(
            'R7 controlled-lineage epoch-1 coefficient is not exactly zero.'
        )
    lineage_smoke_loss.backward()
    lineage_gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            parameter.grad
            for name, parameter in model.named_parameters()
            if name.startswith(prefix) and parameter.grad is not None
        ]
        lineage_gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(gradient).all().item())
            for gradient in gradients
        ) and any(
            float(gradient.detach().float().norm().cpu()) > 0.0
            for gradient in gradients
        )
    model.zero_grad(set_to_none=True)
    if not all(lineage_gradient_checks.values()):
        failed = [
            name for name, passed in lineage_gradient_checks.items()
            if not passed
        ]
        raise RuntimeError(
            'R7 controlled-lineage gradient smoke failed: '
            + ', '.join(failed)
        )

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'architecture': MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82, 'heldout_subjects': 20,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'r7_last_checkpoint_only',
        'strict_best_checkpoint_policy': (
            'save every strict numeric pilot-score improvement'
        ),
        'minimum_delta_policy': (
            'used only to reset early-stopping patience'
        ),
        'r7_preflight_gate_sha256': file_sha256(R7_PREFLIGHT_GATE_PATH),
        'r7_proposal_loss_policy_sha256': file_sha256(R7_POLICY_PATH),
        'component_loss_start_epoch': R7_COMPONENT_LOSS_START_EPOCH,
        'component_loss_full_epoch': R7_COMPONENT_LOSS_FULL_EPOCH,
        'component_loss_maximum_coefficient': (
            R7_COMPONENT_LOSS_MAX_COEFFICIENT
        ),
        'component_positive_weight': R7_POSITIVE_WEIGHT,
        'component_hard_negative_weight': R7_HARD_NEGATIVE_WEIGHT,
        'component_rank_weight': R7_RANK_WEIGHT,
        'component_inner_radius_mm': R7_INNER_RADIUS_MM,
        'component_outer_radius_mm': R7_OUTER_RADIUS_MM,
        'component_logit_margin': R7_LOGIT_MARGIN,
        'component_eligible_tiers': sorted(R7_SMALL_COMPONENT_TIERS),
        'component_coefficient_schedule': {
            str(epoch): coefficient
            for epoch, coefficient in R7_COMPONENT_COEFFICIENT_SCHEDULE.items()
        },
        'proposal_volume_budget_multiplier': R7_VOLUME_BUDGET_MULTIPLIER,
        'proposal_volume_budget_policy': (
            'diagnostic only: retain the top 3x ground-truth target voxels; '
            'never used as an inference rule or checkpoint-selection input'
        ),
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'controlled_lineage_matches_r5_seed': True,
        'controlled_lineage_epoch1_component_coefficient': 0.0,
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(fold0_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            fold0_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'gradient_checks': gradient_checks,
        'controlled_lineage_gradient_checks': lineage_gradient_checks,
        'controlled_lineage_smoke_loss': float(
            lineage_smoke_loss.detach().cpu()
        ),
        'r7_selected_component_smoke_checks': r7_smoke_checks,
        'r7_component_coefficient_schedule_verified': True,
        'r7_smoke_subject_id': smoke_subject_id,
        'r7_smoke_schedule_index': smoke_index,
        'r7_smoke_dropout_flags': smoke_dropout_flags,
        'r5_read_only_baseline_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_volume_bounded_baseline_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r5_checkpoint_used_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R7 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R7 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            '\nNo r7 checkpoint exists. Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(f'Fit / held-out subjects:             82 / 20')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Controlled r5-seed lineage:          verified')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('Local contrastive rescue:              small tiers, epochs 8-40')
    print('Maximum contrastive coefficient:       0.10')
    print('Proposal anti-shortcut diagnostics:    oracle-volume-bounded (audit only)')
    print('R5 recomputed mean/global Dice:        '
          f'{r5_recomputed_summary["positive_mean_dice"]:.4f} / '
          f'{r5_recomputed_summary["positive_global_dice"]:.4f}')
    print('R5 oracle-budgeted small within 8 mm:  '
          f'{r5_recomputed_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}')
    print('Best-checkpoint minimum delta:        disabled (strict numeric best)')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        strict_best_improved = current_score > best_score
        patience_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if strict_best_improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
        if patience_improved:
            patience_reference_score = current_score
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        epoch_tier_metrics = {
            str(row['component_tier']): row for row in tier_rows
        }
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'proposal_voxel_precision': summary['proposal_voxel_precision'],
            'proposal_predicted_to_target_volume_ratio': summary[
                'proposal_predicted_to_target_volume_ratio'
            ],
            'proposal_voxel_average_precision_256bin': summary[
                'proposal_voxel_average_precision_256bin'
            ],
            'budgeted_proposal_component_recall': summary[
                'budgeted_proposal_component_recall'
            ],
            'small_component_proposal_within_2mm_fraction': summary[
                'small_component_proposal_within_2mm_fraction'
            ],
            'small_component_proposal_within_4mm_fraction': summary[
                'small_component_proposal_within_4mm_fraction'
            ],
            'small_component_proposal_within_8mm_fraction': summary[
                'small_component_proposal_within_8mm_fraction'
            ],
            'small_component_budgeted_proposal_within_8mm_fraction': summary[
                'small_component_budgeted_proposal_within_8mm_fraction'
            ],
            'small_component_representation_failure_fraction': summary[
                'small_component_representation_failure_fraction'
            ],
            **{
                f'{tier_name}_{metric_name}': float(
                    epoch_tier_metrics.get(tier_name, {}).get(
                        metric_name, float('nan')
                    )
                )
                for tier_name in COMPONENT_TIERS
                for metric_name in (
                    'final_component_recall',
                    'proposal_component_recall',
                    'budgeted_proposal_component_recall',
                    'proposal_within_8mm_fraction',
                    'budgeted_proposal_within_8mm_fraction',
                    'representation_failure_fraction',
                )
            },
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'strict_best_improved': strict_best_improved,
            'patience_improved_by_minimum_delta': patience_improved,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if strict_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'patience_reference_score': patience_reference_score,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'contrastive={running["local_contrastive_loss"]/max(processed,1):.4f}, '
            f'coef={running["component_loss_coefficient"]/max(processed,1):.3f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        print(
            '  R7 raw proposal geometry: small within 2/4/8 mm='
            f'{summary["small_component_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_proposal_within_8mm_fraction"]:.4f}, '
            'small representation failure='
            f'{summary["small_component_representation_failure_fraction"]:.4f}'
        )
        print(
            '  Oracle-volume-bounded diagnostic: small within 2/4/8 mm='
            f'{summary["small_component_budgeted_proposal_within_2mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_4mm_fraction"]:.4f}/'
            f'{summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}, '
            f'component recall={summary["budgeted_proposal_component_recall"]:.4f}, '
            f'precision/AP={summary["proposal_voxel_precision"]:.4f}/'
            f'{summary["proposal_voxel_average_precision_256bin"]:.4f}, '
            f'volume ratio={summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_baseline = r7_policy['r5_baseline']
    success_policy = r7_policy['r7_directional_success_gates']
    best_tier_by_name = {
        str(row['component_tier']): row for row in best_tier_rows
    }
    best_large_proposal_recall = float(
        best_tier_by_name.get('large', {}).get(
            'proposal_component_recall', 0.0
        )
    )
    mean_dice_improvement = (
        float(best_summary['positive_mean_dice'])
        - float(r5_recomputed_summary['positive_mean_dice'])
    )
    budgeted_small_within_8mm_improvement = (
        float(best_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
        - float(r5_recomputed_summary[
            'small_component_budgeted_proposal_within_8mm_fraction'
        ])
    )
    small_representation_failure_reduction = (
        float(r5_recomputed_summary[
            'small_component_representation_failure_fraction'
        ])
        - float(best_summary[
            'small_component_representation_failure_fraction'
        ])
    )
    global_dice_regression = (
        float(r5_recomputed_summary['positive_global_dice'])
        - float(best_summary['positive_global_dice'])
    )
    proposal_dice_regression = (
        float(r5_recomputed_summary['proposal_global_dice'])
        - float(best_summary['proposal_global_dice'])
    )
    r7_success_checks = {
        'mean_dice_does_not_materially_regress': (
            -mean_dice_improvement
            <= float(success_policy[
                'mean_dice_maximum_allowed_regression'
            ])
        ),
        'mean_dice_reaches_target_improvement': (
            mean_dice_improvement
            >= float(success_policy[
                'mean_dice_target_absolute_improvement'
            ])
        ),
        'volume_bounded_small_within_8mm_improves': (
            budgeted_small_within_8mm_improvement
            >= float(success_policy[
                'volume_bounded_small_within_8mm_absolute_improvement_at_least'
            ])
        ),
        'small_representation_failure_reduced_by_required_margin': (
            small_representation_failure_reduction
            >= float(success_policy[
                'small_representation_failure_absolute_reduction_at_least'
            ])
        ),
        'proposal_component_recall_meets_minimum': (
            float(best_summary['proposal_component_recall'])
            >= float(success_policy['proposal_component_recall_minimum'])
        ),
        'global_dice_regression_within_limit': (
            global_dice_regression
            <= float(success_policy[
                'global_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_dice_regression_within_limit': (
            proposal_dice_regression
            <= float(success_policy[
                'proposal_dice_maximum_allowed_regression'
            ])
        ),
        'proposal_volume_ratio_is_bounded': (
            float(best_summary[
                'proposal_predicted_to_target_volume_ratio'
            ])
            <= float(success_policy[
                'proposal_predicted_to_target_volume_ratio_maximum'
            ])
        ),
        'empty_false_positive_volume_within_limit': (
            float(best_summary['empty_false_positive_volume_ml'])
            <= float(success_policy[
                'mean_false_positive_volume_ml_maximum'
            ])
        ),
        'large_component_proposal_recall_meets_minimum': (
            best_large_proposal_recall
            >= float(success_policy[
                'large_component_proposal_recall_minimum'
            ])
        ),
    }
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'pilot_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'r5_baseline_evidence': r5_baseline,
        'r5_recomputed_under_r7_metric_contract': r5_recomputed_summary,
        'r5_read_only_baseline_diagnostic_path': str(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        'r7_directional_success_policy': success_policy,
        'r7_directional_success_checks': r7_success_checks,
        'r7_directional_success_passed': all(r7_success_checks.values()),
        'r7_directional_effects': {
            'mean_dice_absolute_change': mean_dice_improvement,
            'volume_bounded_small_within_8mm_absolute_change': (
                budgeted_small_within_8mm_improvement
            ),
            'small_representation_failure_absolute_reduction': (
                small_representation_failure_reduction
            ),
            'global_dice_regression': global_dice_regression,
            'proposal_dice_regression': proposal_dice_regression,
            'proposal_predicted_to_target_volume_ratio': float(
                best_summary['proposal_predicted_to_target_volume_ratio']
            ),
            'proposal_voxel_average_precision_256bin': float(
                best_summary['proposal_voxel_average_precision_256bin']
            ),
            'large_component_proposal_recall': (
                best_large_proposal_recall
            ),
        },
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'recommended_next_action': recommendation,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'manual_review_required_before_next_experiment': True,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'full_fivefold_training_authorized': False,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 R7 LOCAL-CONTRASTIVE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(
        'Raw small proposal within 8 mm:   '
        f'{best_summary["small_component_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Oracle-budgeted small within 8 mm: '
        f'{best_summary["small_component_budgeted_proposal_within_8mm_fraction"]:.4f}'
    )
    print(
        'Proposal precision / AP:          '
        f'{best_summary["proposal_voxel_precision"]:.4f} / '
        f'{best_summary["proposal_voxel_average_precision_256bin"]:.4f}'
    )
    print(
        'Proposal predicted/target ratio:  '
        f'{best_summary["proposal_predicted_to_target_volume_ratio"]:.3f}'
    )
    print(
        'Mean Dice change vs recomputed r5:'
        f' {mean_dice_improvement:+.4f}'
    )
    print(
        'Budgeted within-8-mm change vs r5:'
        f' {budgeted_small_within_8mm_improvement:+.4f}'
    )
    print(
        'Small representation failure:     '
        f'{best_summary["small_component_representation_failure_fraction"]:.4f}'
    )
    print(
        'R7 directional success gates:     '
        f'{all(r7_success_checks.values())}'
    )
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')






R7D_EXPOSURE_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure.csv'
R7D_EXPOSURE_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'component_schedule_exposure_by_tier.csv'
)
R7D_RESOLUTION_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_component_audit.csv'
)
R7D_RESOLUTION_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'proposal_resolution_summary.csv'
)
R7D_PAIRED_CASE_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_case_metrics.csv'
)
R7D_PAIRED_COMPONENT_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_paired_component_metrics.csv'
)
R7D_MARGIN_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_train_heldout_margin_audit.csv'
)
R7D_MARGIN_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'r5_r7_margin_summary.csv'
)
R7D_VOLUME_BUDGET_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_audit.csv'
)
R7D_VOLUME_BUDGET_SUMMARY_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'predicted_volume_budget_summary.csv'
)
R7D_CHECKPOINT_PARETO_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'checkpoint_metric_pareto.csv'
)
R7D_BRANCH_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'branch_reliance_comparison.csv'
)
R7D_VISUAL_CANDIDATES_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'visual_qa_candidates.csv'
)
R7D_CHECKS_PATH = R7_FAILURE_DIAGNOSTIC_ROOT / 'diagnostic_checks.csv'
R7D_TARGET_ACCESS_PATH = (
    R7_FAILURE_DIAGNOSTIC_ROOT / 'development_target_access_audit.csv'
)

R7D_OUTPUT_PATHS = [
    R7D_EXPOSURE_PATH,
    R7D_EXPOSURE_SUMMARY_PATH,
    R7D_RESOLUTION_PATH,
    R7D_RESOLUTION_SUMMARY_PATH,
    R7D_PAIRED_CASE_PATH,
    R7D_PAIRED_COMPONENT_PATH,
    R7D_MARGIN_PATH,
    R7D_MARGIN_SUMMARY_PATH,
    R7D_VOLUME_BUDGET_PATH,
    R7D_VOLUME_BUDGET_SUMMARY_PATH,
    R7D_CHECKPOINT_PARETO_PATH,
    R7D_BRANCH_PATH,
    R7D_VISUAL_CANDIDATES_PATH,
    R7D_CHECKS_PATH,
    R7D_TARGET_ACCESS_PATH,
    R7_FAILURE_SUMMARY_PATH,
]


def r7d_existing_result_is_complete():
    if not R7_FAILURE_GATE_PATH.exists():
        return False
    gate = load_json(R7_FAILURE_GATE_PATH)
    if not (
        gate.get('diagnostic_complete') is True
        and gate.get('diagnostic_version') == R7_FAILURE_DIAGNOSTIC_VERSION
        and gate.get('code_revision') == R7_FAILURE_DIAGNOSTIC_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_checkpoint_sha256')
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    ):
        return False
    output_hashes = gate.get('output_sha256', {})
    return all(
        path.exists()
        and output_hashes.get(path.name) == file_sha256(path)
        for path in R7D_OUTPUT_PATHS
    )


def r7d_gini(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0 or float(values.sum()) <= 0.0:
        return 0.0
    values = np.sort(values)
    index = np.arange(1, values.size + 1, dtype=np.float64)
    return float(
        np.sum((2.0 * index - values.size - 1.0) * values)
        / (values.size * values.sum())
    )


def r7d_component_support(row):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    mode = str(row['trial8_supervision_mode'])
    if mode == 'hard_and_soft':
        support = binary
    elif mode == 'soft_only_subvoxel':
        support = soft > 0
    elif mode == 'true_empty':
        support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unsupported supervision mode {mode!r}.'
        )
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: non-finite target arrays.')
    return binary, soft, support, spacing


def r7d_component_schedule_exposure():
    counts = defaultdict(int)
    first_epoch = {}
    last_epoch = {}
    for epoch in range(R7_COMPONENT_LOSS_START_EPOCH, MAX_EPOCHS + 1):
        schedule = pilot_build_epoch_schedule(epoch)
        for entry in schedule:
            if (
                str(entry['sample_type']) != 'positive'
                or str(entry['component_tier']) not in R7_SMALL_COMPONENT_TIERS
            ):
                continue
            key = (
                str(entry['subject_id']), int(entry['atomic_component_id'])
            )
            counts[key] += 1
            first_epoch.setdefault(key, int(epoch))
            last_epoch[key] = int(epoch)

    rows = []
    for item in component_df.sort_values(
        ['subject_id', 'atomic_component_id']
    ).to_dict('records'):
        key = (str(item['subject_id']), int(item['atomic_component_id']))
        tier = str(item['component_tier'])
        eligible = tier in R7_SMALL_COMPONENT_TIERS
        selections = int(counts.get(key, 0)) if eligible else 0
        rows.append({
            'subject_id': key[0],
            'atomic_component_id': key[1],
            'component_tier': tier,
            'component_voxels': int(item['component_voxels']),
            'component_volume_ml': float(item['component_volume_ml']),
            'global_proposal_voxels': int(item['global_proposal_voxels']),
            'local_contrastive_eligible': bool(eligible),
            'epochs_eligible': (
                MAX_EPOCHS - R7_COMPONENT_LOSS_START_EPOCH + 1
                if eligible else 0
            ),
            'selections_epochs_8_to_40': selections,
            'first_selected_epoch': first_epoch.get(key, np.nan),
            'last_selected_epoch': last_epoch.get(key, np.nan),
            'never_selected': bool(eligible and selections == 0),
            'selected_once': bool(eligible and selections == 1),
            'selected_multiple_times': bool(eligible and selections > 1),
        })
    frame = pd.DataFrame(rows)
    summary_rows = []
    for tier in COMPONENT_TIERS:
        group = frame[frame['component_tier'].astype(str) == tier]
        eligible = group[group['local_contrastive_eligible'].astype(bool)]
        values = eligible['selections_epochs_8_to_40'].to_numpy(
            dtype=np.float64
        )
        summary_rows.append({
            'component_tier': tier,
            'components': int(len(group)),
            'eligible_components': int(len(eligible)),
            'never_selected_components': int(
                eligible['never_selected'].sum()
            ),
            'never_selected_fraction': float(
                eligible['never_selected'].mean()
            ) if len(eligible) else 0.0,
            'selected_once_fraction': float(
                eligible['selected_once'].mean()
            ) if len(eligible) else 0.0,
            'minimum_selections': int(values.min()) if len(values) else 0,
            'median_selections': float(np.median(values)) if len(values) else 0.0,
            'mean_selections': float(values.mean()) if len(values) else 0.0,
            'maximum_selections': int(values.max()) if len(values) else 0,
            'selection_gini': r7d_gini(values),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(frame, R7D_EXPOSURE_PATH)
    atomic_write_csv(summary, R7D_EXPOSURE_SUMMARY_PATH)
    return frame, summary


def r7d_maximum_component_cell_matching(cell_sets):
    matched_cell = {}

    def visit(component_index, seen):
        for cell_id in sorted(cell_sets[component_index]):
            if cell_id in seen:
                continue
            seen.add(cell_id)
            if (
                cell_id not in matched_cell
                or visit(matched_cell[cell_id], seen)
            ):
                matched_cell[cell_id] = component_index
                return True
        return False

    matched = 0
    for component_index in range(len(cell_sets)):
        matched += int(visit(component_index, set()))
    return int(matched)


def r7d_resolution_audit():
    component_rows = []
    subject_rows = []
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    canvas_shape = np.asarray(GLOBAL_CANVAS_SHAPE, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    global_shape = tuple(int(v) for v in GLOBAL_INPUT_SHAPE)
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            if partition == 'fold0_fit':
                expected = len(component_records.get(subject_id, []))
                if int(count) != int(expected):
                    raise RuntimeError(
                        f'{subject_id}: resolution audit reconstructed '
                        f'{count} components; frozen audit has {expected}.'
                    )
            pad_start = (canvas_shape - np.asarray(support.shape)) // 2
            cell_sets = []
            component_metadata = []
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for component_id in range(1, int(count) + 1):
                coordinates = np.argwhere(labels == component_id)
                pooled = np.unique(
                    (coordinates + pad_start) // downsample, axis=0
                )
                cell_ids = set(
                    np.ravel_multi_index(pooled.T, global_shape).tolist()
                )
                cell_sets.append(cell_ids)
                voxels = int(len(coordinates))
                volume_ml = voxels * voxel_volume_ml
                component_metadata.append((voxels, volume_ml, pooled))
            cell_frequency = defaultdict(int)
            signature_frequency = defaultdict(int)
            for cells in cell_sets:
                for cell_id in cells:
                    cell_frequency[cell_id] += 1
                signature_frequency[tuple(sorted(cells))] += 1
            matched = r7d_maximum_component_cell_matching(cell_sets)
            collision_components = 0
            duplicate_signature_components = 0
            for component_index, cells in enumerate(cell_sets):
                voxels, volume_ml, pooled = component_metadata[component_index]
                shared_cells = sum(
                    int(cell_frequency[cell_id] > 1) for cell_id in cells
                )
                duplicate_signature = (
                    signature_frequency[tuple(sorted(cells))] > 1
                )
                collision = shared_cells > 0
                collision_components += int(collision)
                duplicate_signature_components += int(duplicate_signature)
                component_rows.append({
                    'development_partition': partition,
                    'subject_id': subject_id,
                    'component_id': int(component_index + 1),
                    'component_tier': pilot_component_tier(volume_ml, voxels),
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'global_proposal_voxels': int(len(cells)),
                    'single_global_voxel': bool(len(cells) == 1),
                    'shared_global_voxels': int(shared_cells),
                    'shares_any_global_voxel': bool(collision),
                    'duplicate_pooled_signature': bool(duplicate_signature),
                    'native_voxels_per_global_voxel': float(
                        voxels / max(len(cells), 1)
                    ),
                })
            subject_rows.append({
                'development_partition': partition,
                'subject_id': subject_id,
                'components': int(count),
                'distinct_global_voxels_used': int(len(cell_frequency)),
                'components_sharing_any_global_voxel': int(collision_components),
                'components_with_duplicate_pooled_signature': int(
                    duplicate_signature_components
                ),
                'maximum_distinct_component_assignment': int(matched),
                'distinct_assignment_ceiling_fraction': float(
                    matched / max(int(count), 1)
                ) if count else 1.0,
            })
            if case_index in {1, len(frame)} or case_index % 20 == 0:
                print(
                    f'  Resolution {partition} {case_index:02d}/{len(frame)}: '
                    f'{subject_id}, components={count}'
                )
    components = pd.DataFrame(component_rows)
    subjects = pd.DataFrame(subject_rows)
    summary_rows = []
    for partition in ['fold0_fit', 'fold0_heldout']:
        for tier in COMPONENT_TIERS:
            group = components[
                (components['development_partition'] == partition)
                & (components['component_tier'] == tier)
            ]
            summary_rows.append({
                'development_partition': partition,
                'component_tier': tier,
                'components': int(len(group)),
                'single_global_voxel_fraction': float(
                    group['single_global_voxel'].mean()
                ) if len(group) else 0.0,
                'global_voxel_collision_fraction': float(
                    group['shares_any_global_voxel'].mean()
                ) if len(group) else 0.0,
                'duplicate_pooled_signature_fraction': float(
                    group['duplicate_pooled_signature'].mean()
                ) if len(group) else 0.0,
                'median_global_proposal_voxels': float(
                    group['global_proposal_voxels'].median()
                ) if len(group) else 0.0,
                'mean_distinct_assignment_ceiling_fraction': float('nan'),
            })
    for partition, group in subjects.groupby('development_partition'):
        summary_rows.append({
            'development_partition': str(partition),
            'component_tier': 'subject_level_distinct_assignment_ceiling',
            'components': int(group['components'].sum()),
            'single_global_voxel_fraction': float('nan'),
            'global_voxel_collision_fraction': float('nan'),
            'duplicate_pooled_signature_fraction': float('nan'),
            'median_global_proposal_voxels': float('nan'),
            'mean_distinct_assignment_ceiling_fraction': float(
                group['distinct_assignment_ceiling_fraction'].mean()
            ),
        })
    summary = pd.DataFrame(summary_rows)
    atomic_write_csv(components, R7D_RESOLUTION_PATH)
    atomic_write_csv(summary, R7D_RESOLUTION_SUMMARY_PATH)
    return components, subjects, summary


def r7d_validation_artifacts(epoch):
    root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    paths = {
        'case': root / 'case_metrics.csv',
        'component': root / 'component_metrics.csv',
        'tier': root / 'component_tier_metrics.csv',
        'summary': root / 'validation_summary.json',
    }
    missing = [str(path) for path in paths.values() if not path.exists()]
    if missing:
        raise RuntimeError(
            'Completed validation artifacts are missing: ' + ', '.join(missing)
        )
    return paths


def r7d_paired_completed_metrics(r7_best_epoch):
    r5_paths = r7d_validation_artifacts(0)
    r7_paths = r7d_validation_artifacts(r7_best_epoch)
    r5_case = pd.read_csv(r5_paths['case'])
    r7_case = pd.read_csv(r7_paths['case'])
    case_keys = ['subject_id']
    paired_case = r5_case.merge(
        r7_case, on=case_keys, how='outer', suffixes=('_r5', '_r7'),
        validate='one_to_one', indicator=True,
    )
    if not bool((paired_case['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 case metrics do not pair one-to-one.')
    paired_case = paired_case.drop(columns=['_merge'])
    case_delta_columns = [
        'dice', 'voxel_precision', 'voxel_recall',
        'predicted_volume_ml_from_mask', 'predicted_volume_ml_from_router',
        'presence_probability', 'proposal_voxel_precision',
        'proposal_voxel_recall', 'proposal_predicted_to_target_volume_ratio',
    ]
    for name in case_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_case and right in paired_case:
            paired_case[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_case[right], errors='coerce')
                - pd.to_numeric(paired_case[left], errors='coerce')
            )

    r5_component = pd.read_csv(r5_paths['component'])
    r7_component = pd.read_csv(r7_paths['component'])
    component_keys = ['subject_id', 'component_id']
    paired_component = r5_component.merge(
        r7_component, on=component_keys, how='outer',
        suffixes=('_r5', '_r7'), validate='one_to_one', indicator=True,
    )
    if not bool((paired_component['_merge'] == 'both').all()):
        raise RuntimeError('R5 and r7 component metrics do not pair one-to-one.')
    paired_component = paired_component.drop(columns=['_merge'])
    component_delta_columns = [
        'final_probability_maximum', 'proposal_probability_maximum',
        'final_maximum_component_iou', 'proposal_maximum_component_iou',
        'proposal_minimum_distance_mm',
        'budgeted_proposal_maximum_component_iou',
        'budgeted_proposal_minimum_distance_mm',
    ]
    for name in component_delta_columns:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                pd.to_numeric(paired_component[right], errors='coerce')
                - pd.to_numeric(paired_component[left], errors='coerce')
            )
    for name in [
        'final_detected', 'proposal_covered',
        'budgeted_proposal_within_8mm',
        'representation_failure_below_0p10',
    ]:
        left, right = f'{name}_r5', f'{name}_r7'
        if left in paired_component and right in paired_component:
            paired_component[f'{name}_delta_r7_minus_r5'] = (
                paired_component[right].map(as_bool).astype(int)
                - paired_component[left].map(as_bool).astype(int)
            )
    atomic_write_csv(paired_case, R7D_PAIRED_CASE_PATH)
    atomic_write_csv(paired_component, R7D_PAIRED_COMPONENT_PATH)

    candidates = []
    positive_case_candidates = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    for label, ascending in [
        ('largest_case_dice_regression', True),
        ('largest_case_dice_improvement', False),
    ]:
        ranked = positive_case_candidates.sort_values(
            'dice_delta_r7_minus_r5', ascending=ascending
        ).head(5)
        for rank, item in enumerate(ranked.to_dict('records'), start=1):
            candidates.append({
                'candidate_reason': label,
                'rank': rank,
                'subject_id': str(item['subject_id']),
                'component_id': np.nan,
                'component_tier': '',
                'r5_value': float(item['dice_r5']),
                'r7_value': float(item['dice_r7']),
                'delta_r7_minus_r5': float(
                    item['dice_delta_r7_minus_r5']
                ),
            })
    small = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ].copy()
    small = small.sort_values(
        'proposal_probability_maximum_delta_r7_minus_r5', ascending=True
    ).head(10)
    for rank, item in enumerate(small.to_dict('records'), start=1):
        candidates.append({
            'candidate_reason': 'largest_small_proposal_probability_regression',
            'rank': rank,
            'subject_id': str(item['subject_id']),
            'component_id': int(item['component_id']),
            'component_tier': str(item['component_tier_r7']),
            'r5_value': float(item['proposal_probability_maximum_r5']),
            'r7_value': float(item['proposal_probability_maximum_r7']),
            'delta_r7_minus_r5': float(
                item['proposal_probability_maximum_delta_r7_minus_r5']
            ),
        })
    candidate_frame = pd.DataFrame(candidates)
    atomic_write_csv(candidate_frame, R7D_VISUAL_CANDIDATES_PATH)
    return paired_case, paired_component, candidate_frame


def r7d_checkpoint_pareto(r7_best_epoch):
    history = pd.read_csv(PILOT_HISTORY_PATH).sort_values('epoch').copy()
    maximize = [
        'validation_mean_dice',
        'validation_global_dice',
        'validation_small_mean_dice',
        'small_component_budgeted_proposal_within_8mm_fraction',
        'proposal_voxel_average_precision_256bin',
    ]
    maximize = [name for name in maximize if name in history.columns]
    values = history[maximize].apply(pd.to_numeric, errors='coerce').fillna(
        -np.inf
    ).to_numpy(dtype=np.float64)
    pareto = np.ones(len(history), dtype=bool)
    for index in range(len(history)):
        dominated = np.any(
            np.all(values >= values[index], axis=1)
            & np.any(values > values[index], axis=1)
        )
        pareto[index] = not bool(dominated)
    history['pareto_nondominated'] = pareto
    history['is_selected_best_epoch'] = (
        history['epoch'].astype(int) == int(r7_best_epoch)
    )
    best_rows = {}
    metric_flags = {}
    for name in maximize + ['pilot_checkpoint_score']:
        if name not in history:
            continue
        values_series = pd.to_numeric(history[name], errors='coerce')
        best_index = values_series.idxmax()
        best_epoch = int(history.loc[best_index, 'epoch'])
        best_value = float(values_series.loc[best_index])
        flag = f'is_best_{name}'
        history[flag] = history['epoch'].astype(int) == best_epoch
        best_rows[name] = {'epoch': best_epoch, 'value': best_value}
        metric_flags[name] = flag
    atomic_write_csv(history, R7D_CHECKPOINT_PARETO_PATH)
    return history, best_rows


def r7d_branch_comparison(r5_best_epoch, r7_best_epoch):
    sources = [
        ('r5', R5_PILOT_ROOT / 'branch_reliance_history.csv', r5_best_epoch),
        ('r7', PILOT_BRANCH_HISTORY_PATH, r7_best_epoch),
    ]
    selected_frames = []
    for checkpoint, path, epoch in sources:
        if not path.exists():
            continue
        frame = pd.read_csv(path)
        available_epochs = sorted(frame['epoch'].astype(int).unique().tolist())
        if not available_epochs:
            continue
        selected_epoch = (
            int(epoch) if int(epoch) in available_epochs
            else min(available_epochs, key=lambda value: abs(value - int(epoch)))
        )
        selected = frame[frame['epoch'].astype(int) == selected_epoch].copy()
        selected['checkpoint'] = checkpoint
        selected['requested_best_epoch'] = int(epoch)
        selected['selected_probe_epoch'] = int(selected_epoch)
        selected['exact_best_epoch_probe'] = bool(selected_epoch == int(epoch))
        selected_frames.append(selected)
    if not selected_frames:
        comparison = pd.DataFrame(columns=[
            'checkpoint', 'ablation', 'mean_target_probability_delta'
        ])
    else:
        raw = pd.concat(selected_frames, ignore_index=True)
        rows = []
        for (checkpoint, ablation), group in raw.groupby(
            ['checkpoint', 'ablation']
        ):
            rows.append({
                'checkpoint': str(checkpoint),
                'ablation': str(ablation),
                'subjects': int(group['subject_id'].nunique()),
                'requested_best_epoch': int(
                    group['requested_best_epoch'].iloc[0]
                ),
                'selected_probe_epoch': int(
                    group['selected_probe_epoch'].iloc[0]
                ),
                'exact_best_epoch_probe': bool(
                    group['exact_best_epoch_probe'].iloc[0]
                ),
                'mean_target_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_target_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_local_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_local_probability_delta'],
                    errors='coerce',
                ).mean()),
                'mean_proposal_probability_delta': float(pd.to_numeric(
                    group['mean_absolute_proposal_probability_delta'],
                    errors='coerce',
                ).mean()),
            })
        comparison = pd.DataFrame(rows)
        if set(comparison['checkpoint']) == {'r5', 'r7'}:
            pivot = comparison.pivot(
                index='ablation', columns='checkpoint',
                values='mean_target_probability_delta',
            )
            deltas = (pivot['r7'] - pivot['r5']).to_dict()
            comparison['target_delta_change_r7_minus_r5'] = comparison[
                'ablation'
            ].map(deltas)
    atomic_write_csv(comparison, R7D_BRANCH_PATH)
    return comparison


def r7d_global_component_masks(labels, count, pad_start):
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    pad = np.asarray(pad_start, dtype=np.int64)
    all_coordinates = np.argwhere(labels > 0)
    full = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    if len(all_coordinates):
        pooled = np.unique((all_coordinates + pad) // downsample, axis=0)
        full[tuple(pooled.T)] = True
    components = []
    for component_id in range(1, int(count) + 1):
        coordinates = np.argwhere(labels == component_id)
        pooled = np.unique((coordinates + pad) // downsample, axis=0)
        components.append(pooled)
    return full, components


def r7d_global_domain(native_shape, pad_start):
    start = np.asarray(pad_start, dtype=np.int64)
    end = start + np.asarray(native_shape, dtype=np.int64)
    downsample = np.asarray(GLOBAL_DOWNSAMPLE, dtype=np.int64)
    low_start = start // downsample
    low_end = (end + downsample - 1) // downsample
    domain = np.zeros(GLOBAL_INPUT_SHAPE, dtype=bool)
    slices = tuple(
        slice(int(low_start[axis]), int(low_end[axis])) for axis in range(3)
    )
    domain[slices] = True
    return domain


def r7d_component_logit_margins(
    logits, full_target, component_coordinates, domain, subject_id,
    partition, checkpoint, spacing,
):
    rows = []
    outer_radius = np.asarray(R7_OUTER_STRUCTURE.shape, dtype=np.int64) // 2
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    native_voxel_volume = voxel_volume_ml
    for component_id, pooled in enumerate(component_coordinates, start=1):
        if len(pooled) == 0:
            continue
        minimum = np.maximum(pooled.min(axis=0) - outer_radius, 0)
        maximum = np.minimum(
            pooled.max(axis=0) + outer_radius + 1,
            np.asarray(GLOBAL_INPUT_SHAPE),
        )
        slices = tuple(
            slice(int(minimum[axis]), int(maximum[axis]))
            for axis in range(3)
        )
        exact = np.zeros(tuple((maximum - minimum).tolist()), dtype=bool)
        local_coordinates = pooled - minimum
        exact[tuple(local_coordinates.T)] = True
        _, _, negative = r7_component_regions(
            exact, full_target[slices], domain[slices]
        )
        positive_logits = logits[slices][exact]
        negative_logits = logits[slices][negative]
        hard_count = min(
            len(negative_logits),
            max(R7_HARD_NEGATIVE_MINIMUM,
                R7_HARD_NEGATIVE_RATIO * len(positive_logits)),
        )
        if hard_count > 0:
            hard_negative = np.partition(
                negative_logits, len(negative_logits) - hard_count
            )[-hard_count:]
            hard_negative_mean = float(hard_negative.mean())
            hard_negative_maximum = float(hard_negative.max())
        else:
            hard_negative_mean = float('nan')
            hard_negative_maximum = float('nan')
        native_voxels_estimate = int(len(pooled) * np.prod(GLOBAL_DOWNSAMPLE))
        volume_ml_estimate = native_voxels_estimate * native_voxel_volume
        positive_mean = float(positive_logits.mean())
        positive_minimum = float(positive_logits.min())
        positive_maximum = float(positive_logits.max())
        rows.append({
            'checkpoint': checkpoint,
            'development_partition': partition,
            'subject_id': subject_id,
            'component_id': int(component_id),
            'component_tier_from_pooled_extent': pilot_component_tier(
                volume_ml_estimate, native_voxels_estimate
            ),
            'global_proposal_voxels': int(len(pooled)),
            'positive_logit_mean': positive_mean,
            'positive_logit_minimum': positive_minimum,
            'positive_logit_maximum': positive_maximum,
            'hard_negative_logit_mean': hard_negative_mean,
            'hard_negative_logit_maximum': hard_negative_maximum,
            'positive_minus_hard_negative_mean_margin': (
                positive_mean - hard_negative_mean
                if np.isfinite(hard_negative_mean) else float('nan')
            ),
            'positive_minimum_minus_hard_negative_maximum_margin': (
                positive_minimum - hard_negative_maximum
                if np.isfinite(hard_negative_maximum) else float('nan')
            ),
            'positive_probability_mean': float(
                1.0 / (1.0 + np.exp(-np.clip(positive_mean, -30, 30)))
            ),
            'negative_annulus_voxels': int(negative.sum()),
            'hard_negative_voxels_used': int(hard_count),
        })
    return rows


def r7d_topk_prediction(probability, budget):
    probability = np.asarray(probability, dtype=np.float32)
    budget = int(np.clip(int(budget), 1, probability.size))
    flat = probability.reshape(-1)
    selected = np.argpartition(flat, flat.size - budget)[-budget:]
    prediction = np.zeros(flat.size, dtype=bool)
    prediction[selected] = True
    return prediction.reshape(probability.shape)


def r7d_budget_metrics(
    probability, target, spacing, predicted_volume_ml, subject_id,
    checkpoint,
):
    target = np.asarray(target, dtype=bool)
    target_voxels = int(target.sum())
    if target_voxels == 0:
        return []
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    labels, count = ndimage.label(
        target, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    component_objects = ndimage.find_objects(labels)
    component_metadata = []
    for component_id, component_slice in enumerate(component_objects, start=1):
        if component_slice is None:
            continue
        local = labels[component_slice] == component_id
        voxels = int(local.sum())
        volume_ml = voxels * voxel_volume_ml
        component_metadata.append((
            component_id, component_slice, local,
            pilot_component_tier(volume_ml, voxels),
        ))
    budgets = [
        ('router_x1', max(1, int(math.ceil(predicted_volume_ml / voxel_volume_ml)))),
        ('router_x2', max(1, int(math.ceil(2.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('router_x3', max(1, int(math.ceil(3.0 * predicted_volume_ml / voxel_volume_ml)))),
        ('oracle_target_x3', max(1, int(math.ceil(3.0 * target_voxels)))),
    ]
    rows = []
    for budget_name, requested_budget in budgets:
        prediction = r7d_topk_prediction(probability, requested_budget)
        predicted_voxels = int(prediction.sum())
        intersection = int((prediction & target).sum())
        distance = ndimage.distance_transform_edt(
            ~prediction, sampling=spacing
        )
        within_8 = []
        small_within_8 = []
        for _, component_slice, component_local, tier in component_metadata:
            minimum_distance = float(
                distance[component_slice][component_local].min(initial=np.inf)
            )
            covered = bool(minimum_distance <= 8.0)
            within_8.append(covered)
            if tier in R7_SMALL_COMPONENT_TIERS:
                small_within_8.append(covered)
        rows.append({
            'checkpoint': checkpoint,
            'subject_id': subject_id,
            'budget_policy': budget_name,
            'router_predicted_volume_ml': float(predicted_volume_ml),
            'target_volume_ml': float(target_voxels * voxel_volume_ml),
            'router_to_target_volume_ratio': float(
                predicted_volume_ml / max(target_voxels * voxel_volume_ml, 1e-8)
            ),
            'requested_budget_voxels': int(requested_budget),
            'selected_budget_voxels': predicted_voxels,
            'selected_to_target_voxel_ratio': float(
                predicted_voxels / max(target_voxels, 1)
            ),
            'voxel_precision': float(intersection / max(predicted_voxels, 1)),
            'voxel_recall': float(intersection / max(target_voxels, 1)),
            'dice': float(
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            ),
            'components': int(len(component_metadata)),
            'component_within_8mm_fraction': float(np.mean(within_8)),
            'small_components': int(len(small_within_8)),
            'small_component_within_8mm_fraction': float(
                np.mean(small_within_8)
            ) if small_within_8 else float('nan'),
        })
    return rows


def r7d_load_model(checkpoint_path, expected_version, expected_revision, device):
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=False
    )
    checks = {
        'pilot_version': checkpoint.get('pilot_version') == expected_version,
        'code_revision': checkpoint.get('code_revision') == expected_revision,
        'parameter_count': int(checkpoint.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS,
        'ema_model_state': 'ema_model_state' in checkpoint,
    }
    if not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            f'Unsafe read-only checkpoint {checkpoint_path}: '
            + ', '.join(failed)
        )
    model = VascuFateNetV22(pilot_schema).to(device)
    model.load_state_dict(checkpoint['ema_model_state'], strict=True)
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    del checkpoint
    return model


@torch.no_grad()
def r7d_checkpoint_margin_and_volume_audit(
    checkpoint_label, checkpoint_path, expected_version,
    expected_revision, device,
):
    model = r7d_load_model(
        checkpoint_path, expected_version, expected_revision, device
    )
    margin_rows = []
    budget_rows = []
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    partitions = [
        ('fold0_fit', fold0_fit_df),
        ('fold0_heldout', fold0_heldout_df),
    ]
    for partition, frame in partitions:
        for case_index, row in enumerate(frame.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            case = pilot_load_validation_case(row)
            batch = pilot_stack_sample(case['base_sample'], device)
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(model, batch)
            proposal_logits_tensor = cache['proposal_logits'].float()
            logits = proposal_logits_tensor[0, 0].cpu().numpy()
            predicted_volume_ml = float(torch.expm1(
                cache['routed']['log_volume_prediction'].float()
            )[0].clamp_min(0).cpu())
            _, _, support, spacing = r7d_component_support(row)
            labels, count = ndimage.label(support, structure=structure)
            full_global, component_coordinates = r7d_global_component_masks(
                labels, count, case['pad_start']
            )
            domain = r7d_global_domain(case['native_shape'], case['pad_start'])
            rows = r7d_component_logit_margins(
                logits, full_global, component_coordinates, domain,
                subject_id, partition, checkpoint_label, spacing,
            )
            
            voxel_volume_ml = float(np.prod(spacing) / 1000.0)
            for item in rows:
                component_id = int(item['component_id'])
                voxels = int((labels == component_id).sum())
                volume_ml = voxels * voxel_volume_ml
                item['component_voxels'] = voxels
                item['component_volume_ml'] = volume_ml
                item['component_tier'] = pilot_component_tier(
                    volume_ml, voxels
                )
            margin_rows.extend(rows)

            if (
                partition == 'fold0_heldout'
                and as_bool(row['trial8_hard_metric_eligible'])
                and bool(case['target'].any())
            ):
                proposal_canvas = F.interpolate(
                    torch.sigmoid(proposal_logits_tensor),
                    size=GLOBAL_CANVAS_SHAPE,
                    mode='trilinear', align_corners=False,
                )[0, 0].cpu().numpy()
                native_slices = tuple(
                    slice(
                        case['pad_start'][axis],
                        case['pad_start'][axis] + case['native_shape'][axis],
                    )
                    for axis in range(3)
                )
                proposal_native = proposal_canvas[native_slices]
                budget_rows.extend(r7d_budget_metrics(
                    proposal_native, case['target'], case['spacing'],
                    predicted_volume_ml, subject_id, checkpoint_label,
                ))
            del batch, cache, proposal_logits_tensor, case
            if case_index in {1, len(frame)} or case_index % 10 == 0:
                print(
                    f'  {checkpoint_label} {partition} '
                    f'{case_index:02d}/{len(frame)}: {subject_id}'
                )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return margin_rows, budget_rows


def r7d_margin_summary(frame):
    rows = []
    for (checkpoint, partition, tier), group in frame.groupby([
        'checkpoint', 'development_partition', 'component_tier'
    ]):
        margin = pd.to_numeric(
            group['positive_minus_hard_negative_mean_margin'],
            errors='coerce',
        )
        rows.append({
            'checkpoint': str(checkpoint),
            'development_partition': str(partition),
            'component_tier': str(tier),
            'components': int(len(group)),
            'mean_logit_margin': float(margin.mean()),
            'median_logit_margin': float(margin.median()),
            'positive_margin_fraction': float((margin > 0).mean()),
            'mean_positive_probability': float(pd.to_numeric(
                group['positive_probability_mean'], errors='coerce'
            ).mean()),
        })
    return pd.DataFrame(rows)


def r7d_volume_budget_summary(frame):
    rows = []
    for (checkpoint, policy), group in frame.groupby(
        ['checkpoint', 'budget_policy']
    ):
        rows.append({
            'checkpoint': str(checkpoint),
            'budget_policy': str(policy),
            'subjects': int(group['subject_id'].nunique()),
            'mean_dice': float(group['dice'].mean()),
            'mean_voxel_precision': float(group['voxel_precision'].mean()),
            'mean_voxel_recall': float(group['voxel_recall'].mean()),
            'mean_component_within_8mm_fraction': float(
                group['component_within_8mm_fraction'].mean()
            ),
            'mean_small_component_within_8mm_fraction': float(
                group['small_component_within_8mm_fraction'].mean()
            ),
            'median_router_to_target_volume_ratio': float(
                group['router_to_target_volume_ratio'].median()
            ),
            'mean_selected_to_target_voxel_ratio': float(
                group['selected_to_target_voxel_ratio'].mean()
            ),
        })
    return pd.DataFrame(rows)


def r7d_small_margin_delta(summary, partition):
    subset = summary[
        (summary['development_partition'] == partition)
        & summary['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    if not len(subset):
        return float('nan')
    weighted = {}
    for checkpoint in ['r5', 'r7']:
        group = subset[subset['checkpoint'] == checkpoint]
        if not len(group):
            weighted[checkpoint] = float('nan')
            continue
        weighted[checkpoint] = float(np.average(
            group['mean_logit_margin'], weights=group['components']
        ))
    return weighted['r7'] - weighted['r5']


def run_r7_failure_mechanism_diagnostic():
    if r7d_existing_result_is_complete():
        gate = load_json(R7_FAILURE_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC ALREADY COMPLETE')
        print('=' * 80)
        print(f'Dominant mechanism: {gate["dominant_failure_mechanism"]}')
        print(f'Diagnostic gate:    {R7_FAILURE_GATE_PATH}')
        print('No source checkpoint or completed diagnostic was rewritten.')
        return

    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r7_best_epoch = int(r7_result['best_epoch'])
    r5_best_epoch = int(r5_result['best_epoch'])
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_sha_before = file_sha256(PILOT_BEST_CHECKPOINT_PATH)
    source_paths = [
        R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH,
        R5_RESULT_GATE_PATH, PILOT_RESULT_GATE_PATH, PILOT_HISTORY_PATH,
    ]
    source_hashes_before = {
        str(path): file_sha256(path) for path in source_paths
    }
    initial_checks = {
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r7_pilot_complete': r7_result.get('pilot_complete') is True,
        'r5_checkpoint_hash_matches_result_gate': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'r7_checkpoint_hash_matches_result_gate': (
            r7_result.get('best_checkpoint_sha256') == r7_sha_before
        ),
        'r7_directional_success_was_false': (
            r7_result.get('r7_directional_success_passed') is False
        ),
        'full_fivefold_training_locked': (
            r7_result.get('full_fivefold_training_authorized') is False
        ),
        'fold0_subject_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(locked_validation_df['subject_id']))
        ),
        'sealed_test_subjects_excluded': (
            set(fold0_fit_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
            and set(fold0_heldout_df['subject_id'])
            .isdisjoint(set(sealed_test_df['subject_id']))
        ),
    }
    if not all(initial_checks.values()):
        failed = [name for name, passed in initial_checks.items() if not passed]
        raise RuntimeError(
            'R7 failure diagnostic upstream checks failed: '
            + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC STARTING')
    print('=' * 80)
    print(f'R5 / r7 best epochs:               {r5_best_epoch} / {r7_best_epoch}')
    print('Target access:                     fold-0 development only')
    print('Locked validation targets read:    False')
    print('Test targets read:                 False')
    print('Optimizer / backward / steps:      0 / 0 / 0')

    target_access_frame = pd.DataFrame([
        {
            'subject_id': str(row['subject_id']),
            'development_partition': partition,
            'target_arrays_authorized': True,
            'target_arrays_read_by_diagnostic': True,
            'locked_validation_target_read': False,
            'sealed_test_target_read': False,
        }
        for partition, frame in [
            ('fold0_fit', fold0_fit_df),
            ('fold0_heldout', fold0_heldout_df),
        ]
        for row in frame.to_dict('records')
    ])
    atomic_write_csv(target_access_frame, R7D_TARGET_ACCESS_PATH)

    exposure, exposure_summary = r7d_component_schedule_exposure()
    resolution, resolution_subjects, resolution_summary = (
        r7d_resolution_audit()
    )
    paired_case, paired_component, visual_candidates = (
        r7d_paired_completed_metrics(r7_best_epoch)
    )
    checkpoint_pareto, metric_best_rows = r7d_checkpoint_pareto(
        r7_best_epoch
    )
    branch_comparison = r7d_branch_comparison(
        r5_best_epoch, r7_best_epoch
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            'The train-versus-held-out r5/r7 margin audit requires CUDA.'
        )
    set_determinism(PILOT_SEED)
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    all_margin_rows = []
    all_budget_rows = []
    checkpoint_specs = [
        ('r5', R5_BEST_CHECKPOINT_PATH,
         R5_PILOT_VERSION, R5_PILOT_CODE_REVISION),
        ('r7', PILOT_BEST_CHECKPOINT_PATH,
         PILOT_VERSION, PILOT_CODE_REVISION),
    ]
    for label, path, version, revision in checkpoint_specs:
        print(f'\nRead-only global proposal audit: {label}')
        margin_rows, budget_rows = r7d_checkpoint_margin_and_volume_audit(
            label, path, version, revision, device
        )
        all_margin_rows.extend(margin_rows)
        all_budget_rows.extend(budget_rows)
    margins = pd.DataFrame(all_margin_rows)
    budgets = pd.DataFrame(all_budget_rows)
    margin_summary = r7d_margin_summary(margins)
    budget_summary = r7d_volume_budget_summary(budgets)
    atomic_write_csv(margins, R7D_MARGIN_PATH)
    atomic_write_csv(margin_summary, R7D_MARGIN_SUMMARY_PATH)
    atomic_write_csv(budgets, R7D_VOLUME_BUDGET_PATH)
    atomic_write_csv(budget_summary, R7D_VOLUME_BUDGET_SUMMARY_PATH)

    small_exposure = exposure[
        exposure['local_contrastive_eligible'].astype(bool)
    ]
    never_selected_fraction = float(
        small_exposure['never_selected'].mean()
    )
    heldout_small_resolution = resolution[
        (resolution['development_partition'] == 'fold0_heldout')
        & resolution['component_tier'].isin(R7_SMALL_COMPONENT_TIERS)
    ]
    heldout_small_collision_fraction = float(
        heldout_small_resolution['shares_any_global_voxel'].mean()
    ) if len(heldout_small_resolution) else float('nan')
    heldout_assignment_ceiling = float(
        resolution_subjects[
            resolution_subjects['development_partition'] == 'fold0_heldout'
        ]['distinct_assignment_ceiling_fraction'].mean()
    )
    fit_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_fit'
    )
    heldout_small_margin_delta = r7d_small_margin_delta(
        margin_summary, 'fold0_heldout'
    )
    positive_paired_cases = paired_case[
        pd.to_numeric(
            paired_case['target_voxels_r7'], errors='coerce'
        ).fillna(0) > 0
    ]
    case_dice_delta = float(
        positive_paired_cases['dice_delta_r7_minus_r5'].mean()
    )
    small_components = paired_component[
        paired_component['component_tier_r7'].astype(str).isin(
            R7_SMALL_COMPONENT_TIERS
        )
    ]
    small_proposal_probability_delta = float(pd.to_numeric(
        small_components[
            'proposal_probability_maximum_delta_r7_minus_r5'
        ], errors='coerce'
    ).mean())
    r7_budget = budget_summary[
        budget_summary['checkpoint'] == 'r7'
    ]
    router_x3 = r7_budget[
        r7_budget['budget_policy'] == 'router_x3'
    ]
    oracle_x3 = r7_budget[
        r7_budget['budget_policy'] == 'oracle_target_x3'
    ]
    router_small_within8 = float(
        router_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    oracle_small_within8 = float(
        oracle_x3['mean_small_component_within_8mm_fraction'].iloc[0]
    )
    deployment_budget_gap = oracle_small_within8 - router_small_within8
    selected_epoch = int(r7_best_epoch)
    mean_dice_best_epoch = int(
        metric_best_rows['validation_mean_dice']['epoch']
    )
    checkpoint_selection_mismatch = selected_epoch != mean_dice_best_epoch

    mechanism_evidence = {
        'component_exposure_imbalance': bool(never_selected_fraction > 0.10),
        'low_resolution_component_collision': bool(
            heldout_small_collision_fraction > 0.10
            or heldout_assignment_ceiling < 0.95
        ),
        'contrastive_train_heldout_margin_divergence': bool(
            np.isfinite(fit_small_margin_delta)
            and np.isfinite(heldout_small_margin_delta)
            and fit_small_margin_delta > heldout_small_margin_delta + 0.10
        ),
        'small_proposal_signal_regressed': bool(
            small_proposal_probability_delta < 0.0
        ),
        'deployment_volume_router_limits_coverage': bool(
            deployment_budget_gap > 0.10
        ),
        'checkpoint_objective_mismatch': bool(checkpoint_selection_mismatch),
        'r7_mean_dice_regressed': bool(case_dice_delta < 0.0),
        'r7_representation_failure_extreme': bool(
            float(r7_result['best_validation_summary'][
                'small_component_representation_failure_fraction'
            ]) > 0.75
        ),
    }
    if mechanism_evidence['contrastive_train_heldout_margin_divergence']:
        dominant_mechanism = (
            'low-resolution local-contrastive over-specialization with '
            'poor held-out satellite transfer'
        )
    elif mechanism_evidence['component_exposure_imbalance']:
        dominant_mechanism = (
            'incomplete atomic-component exposure in the stochastic curriculum'
        )
    elif mechanism_evidence['low_resolution_component_collision']:
        dominant_mechanism = (
            'global proposal-grid resolution and component-collision ceiling'
        )
    else:
        dominant_mechanism = (
            'proposal supervision/refinement mismatch rather than missing '
            'clinical or arterial routing capacity'
        )
    recommended_r8 = {
        'architecture': (
            'Preserve the r5 V2.2 global/arterial/clinical path; add a bounded '
            'native-resolution local satellite-proposal head and fuse it as '
            'a zero-initialized residual, leaving the coarse global proposal '
            'responsible for burden and broad localization.'
        ),
        'curriculum': (
            'Replace stochastic inverse-size component draws with a '
            'coverage-guaranteed component cycle before repeats, stratified '
            'by subvoxel/micro/tiny/small tier and subject.'
        ),
        'objective': (
            'Apply component contrast at native local resolution, normalize '
            'per component, cap its gradient contribution against the main '
            'segmentation loss, and retain the coarse proposal loss.'
        ),
        'deployment': (
            'Train and calibrate the predicted-volume router on fold-fit data '
            'only; use predicted rather than target volume for any inference '
            'budget and report the oracle budget only as a diagnostic ceiling.'
        ),
        'selection': (
            'Use a Pareto-safe checkpoint rule that requires no mean-Dice '
            'regression and explicit held-out small-component improvement; '
            'do not optimize the present composite score alone.'
        ),
        'next_stage': (
            'Build an R8 read-only architecture/loss preflight first, then one '
            'fold-0 pilot. Five-fold and locked validation/test evaluation '
            'remain unauthorized.'
        ),
    }

    source_hashes_after = {
        str(path): file_sha256(path) for path in source_paths
    }
    final_checks = {
        **initial_checks,
        'fit_component_count_matches_frozen_872': int(
            (resolution['development_partition'] == 'fold0_fit').sum()
        ) == 872,
        'exposure_covers_every_frozen_fit_component': len(exposure)
        == len(component_df),
        'paired_case_metrics_have_20_subjects': len(paired_case) == 20,
        'paired_component_metrics_are_nonempty': len(paired_component) > 0,
        'margin_audit_has_both_checkpoints': set(margins['checkpoint'])
        == {'r5', 'r7'},
        'margin_audit_has_fit_and_heldout': set(
            margins['development_partition']
        ) == {'fold0_fit', 'fold0_heldout'},
        'all_finite_margin_inputs': bool(np.isfinite(pd.to_numeric(
            margins['positive_logit_mean'], errors='coerce'
        )).all()),
        'deployment_budget_uses_heldout_development_only': bool(
            budgets['subject_id'].isin(fold0_heldout_df['subject_id']).all()
        ),
        'development_target_access_has_exactly_102_subjects': (
            len(target_access_frame) == 102
            and target_access_frame['subject_id'].nunique() == 102
        ),
        'development_target_access_excludes_locked_splits': bool(
            (~target_access_frame['locked_validation_target_read']).all()
            and (~target_access_frame['sealed_test_target_read']).all()
        ),
        'r5_checkpoint_unchanged': r5_sha_before
        == file_sha256(R5_BEST_CHECKPOINT_PATH),
        'r7_checkpoint_unchanged': r7_sha_before
        == file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'all_source_artifacts_unchanged': source_hashes_before
        == source_hashes_after,
        'optimizer_not_constructed': True,
        'backward_not_called': True,
        'optimizer_steps_are_zero': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    diagnostic_passed = all(final_checks.values())

    summary = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'diagnostic_version': R7_FAILURE_DIAGNOSTIC_VERSION,
        'code_revision': R7_FAILURE_DIAGNOSTIC_CODE_REVISION,
        'r5_best_epoch': r5_best_epoch,
        'r7_best_epoch': r7_best_epoch,
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_sha256': r7_sha_before,
        'observed_effects': {
            'mean_paired_case_dice_delta_r7_minus_r5': case_dice_delta,
            'small_proposal_probability_delta_r7_minus_r5': (
                small_proposal_probability_delta
            ),
            'eligible_components_never_selected_fraction': (
                never_selected_fraction
            ),
            'heldout_small_global_voxel_collision_fraction': (
                heldout_small_collision_fraction
            ),
            'heldout_distinct_assignment_ceiling_fraction': (
                heldout_assignment_ceiling
            ),
            'fit_small_logit_margin_delta_r7_minus_r5': (
                fit_small_margin_delta
            ),
            'heldout_small_logit_margin_delta_r7_minus_r5': (
                heldout_small_margin_delta
            ),
            'r7_router_x3_small_within_8mm_fraction': router_small_within8,
            'r7_oracle_x3_small_within_8mm_fraction': oracle_small_within8,
            'deployment_to_oracle_small_within_8mm_gap': (
                deployment_budget_gap
            ),
            'selected_checkpoint_epoch': selected_epoch,
            'mean_dice_best_epoch': mean_dice_best_epoch,
        },
        'mechanism_evidence': mechanism_evidence,
        'dominant_failure_mechanism': dominant_mechanism,
        'recommended_r8_contract': recommended_r8,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': False,
        'backward_called': False,
        'optimizer_steps': 0,
        'r8_preflight_implementation_allowed': bool(diagnostic_passed),
        'r8_training_authorized': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(summary, R7_FAILURE_SUMMARY_PATH)
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(value)}
        for name, value in final_checks.items()
    ])
    atomic_write_csv(checks_frame, R7D_CHECKS_PATH)
    output_hashes = {
        path.name: file_sha256(path) for path in R7D_OUTPUT_PATHS
    }
    gate = {
        **summary,
        'diagnostic_complete': bool(diagnostic_passed),
        'diagnostic_passed': bool(diagnostic_passed),
        'checks': final_checks,
        'source_sha256_before': source_hashes_before,
        'source_sha256_after': source_hashes_after,
        'output_sha256': output_hashes,
        'diagnostic_root': str(R7_FAILURE_DIAGNOSTIC_ROOT),
        'summary_path': str(R7_FAILURE_SUMMARY_PATH),
    }
    atomic_write_json(gate, R7_FAILURE_GATE_PATH)
    if not diagnostic_passed:
        failed = [
            name for name, passed in final_checks.items() if not passed
        ]
        raise RuntimeError(
            'TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC FAILED. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nComponent exposure by tier:')
    display(exposure_summary)
    print('\nProposal-grid resolution summary:')
    display(resolution_summary)
    print('\nTrain-versus-held-out proposal-logit margins:')
    display(margin_summary)
    print('\nDeployment-legal predicted-volume budget audit:')
    display(budget_summary)
    print('\nBranch reliance comparison:')
    display(branch_comparison)
    print('\nDiagnostic checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R7 FAILURE-MECHANISM DIAGNOSTIC PASSED')
    print('=' * 80)
    print(f'Dominant mechanism:                 {dominant_mechanism}')
    print(f'Eligible components never selected: {never_selected_fraction:.4f}')
    print(f'Fit small-margin change r7-r5:      {fit_small_margin_delta:+.4f}')
    print(f'Heldout small-margin change r7-r5:  {heldout_small_margin_delta:+.4f}')
    print(f'R7 router-x3 small within 8 mm:     {router_small_within8:.4f}')
    print(f'R7 oracle-x3 small within 8 mm:     {oracle_small_within8:.4f}')
    print(f'Mean-Dice best / selected epoch:    {mean_dice_best_epoch} / {selected_epoch}')
    print('Optimizer / backward / steps:       0 / 0 / 0')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('R8 preflight implementation allowed: True')
    print('R8 training authorized:             False')
    print('Full five-fold training authorized: False')
    print(f'Diagnostic summary:                 {R7_FAILURE_SUMMARY_PATH}')
    print(f'Diagnostic gate:                    {R7_FAILURE_GATE_PATH}')
    print(
        '\nNext: construct the read-only R8 dual-resolution satellite-proposal '
        'and component-cycled curriculum preflight; do not train yet.'
    )


run_r7_failure_mechanism_diagnostic()






R8_MODEL_VERSION = 'trial8_vascufate_net_v2_3_dual_resolution_satellite'
R8_PREFLIGHT_VERSION = 'v1_r5_warmstart_component_cycle_native_satellite'
R8_CODE_REVISION = (
    'r1_native_component_absolute_calibration_pareto_guard'
)
R8_SEED = 20261007
R8_SATELLITE_GATE_BOUND = 0.20
R8_SATELLITE_FEATURE_CHANNELS = 16
R8_COMPONENT_POSITIVE_PATCHES = 112
R8_TOPOLOGY_NEGATIVE_PATCHES = 28
R8_BRAIN_NEGATIVE_PATCHES = 28
R8_SCHEDULE_EPOCHS = 40
R8_NATIVE_INNER_RADIUS_MM = 2.0
R8_NATIVE_OUTER_RADIUS_MM = 12.0
R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR = 1.0
R8_MAXIMUM_NEW_PARAMETERS = 250_000
R8_TIER_QUOTAS = {
    'subvoxel_or_single_voxel': 2,
    'micro': 76,
    'tiny': 27,
    'small': 7,
}

R8_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / R8_MODEL_VERSION / R8_PREFLIGHT_VERSION
)
R8_PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
R8_ARCHITECTURE_PATH = R8_PREFLIGHT_ROOT / 'r8_architecture_config.json'
R8_TRAINING_POLICY_PATH = R8_PREFLIGHT_ROOT / 'r8_training_policy.json'
R8_SCHEDULE_PATH = R8_PREFLIGHT_ROOT / 'r8_component_cycle_schedule.csv'
R8_SCHEDULE_SUMMARY_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_component_cycle_summary.csv'
)
R8_NATIVE_TARGET_AUDIT_PATH = (
    R8_PREFLIGHT_ROOT / 'r8_native_component_target_audit.csv'
)
R8_PARAMETER_PATH = R8_PREFLIGHT_ROOT / 'r8_parameter_audit.csv'
R8_OUTPUT_PATH = R8_PREFLIGHT_ROOT / 'r8_output_shape_audit.csv'
R8_NEUTRALITY_PATH = R8_PREFLIGHT_ROOT / 'r8_neutrality_audit.csv'
R8_GRADIENT_PATH = R8_PREFLIGHT_ROOT / 'r8_gradient_audit.csv'
R8_LOSS_PATH = R8_PREFLIGHT_ROOT / 'r8_loss_smoke_audit.csv'
R8_CHECKS_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_checks.csv'
R8_GATE_PATH = R8_PREFLIGHT_ROOT / 'r8_preflight_gate.json'

R8_OUTPUT_ARTIFACTS = [
    R8_ARCHITECTURE_PATH,
    R8_TRAINING_POLICY_PATH,
    R8_SCHEDULE_PATH,
    R8_SCHEDULE_SUMMARY_PATH,
    R8_NATIVE_TARGET_AUDIT_PATH,
    R8_PARAMETER_PATH,
    R8_OUTPUT_PATH,
    R8_NEUTRALITY_PATH,
    R8_GRADIENT_PATH,
    R8_LOSS_PATH,
    R8_CHECKS_PATH,
]


def r8_existing_preflight_complete():
    if not R8_GATE_PATH.exists():
        return False
    gate = load_json(R8_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('model_version') == R8_MODEL_VERSION
        and gate.get('preflight_version') == R8_PREFLIGHT_VERSION
        and gate.get('code_revision') == R8_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and gate.get('r7_failure_diagnostic_sha256')
        == file_sha256(R7_FAILURE_GATE_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R8_OUTPUT_ARTIFACTS
    )


def r8_tier_records():
    records = {}
    for tier in R8_TIER_QUOTAS:
        frame = component_df[
            component_df['component_tier'].astype(str) == tier
        ].sort_values(['subject_id', 'atomic_component_id'])
        records[tier] = frame.to_dict('records')
        if not records[tier]:
            raise RuntimeError(f'R8 component tier {tier!r} is empty.')
    return records


R8_COMPONENT_RECORDS_BY_TIER = r8_tier_records()


def r8_tier_permutation(tier, cycle_index):
    tier_index = list(R8_TIER_QUOTAS).index(str(tier))
    rng = np.random.default_rng(
        R8_SEED + tier_index * 10_000_019 + int(cycle_index) * 1_000_003
    )
    count = len(R8_COMPONENT_RECORDS_BY_TIER[str(tier)])
    return rng.permutation(count).tolist()


def r8_component_for_position(tier, position):
    records = R8_COMPONENT_RECORDS_BY_TIER[str(tier)]
    cycle_index = int(position) // len(records)
    offset = int(position) % len(records)
    permutation = r8_tier_permutation(tier, cycle_index)
    item = dict(records[int(permutation[offset])])
    item['r8_tier_cycle_index'] = int(cycle_index)
    item['r8_tier_cycle_offset'] = int(offset)
    return item


def r8_build_epoch_schedule(epoch):
    epoch = int(epoch)
    if epoch < 1:
        raise ValueError('R8 epoch must be positive.')
    positives = []
    for tier, quota in R8_TIER_QUOTAS.items():
        for slot in range(int(quota)):
            position = (epoch - 1) * int(quota) + slot
            component = r8_component_for_position(tier, position)
            subject_id = str(component['subject_id'])
            jitter_rng = np.random.default_rng(
                R8_SEED + epoch * 10_000_019
                + list(R8_TIER_QUOTAS).index(tier) * 104_729 + slot
            )
            center = np.asarray([
                component['centroid_x_vox'],
                component['centroid_y_vox'],
                component['centroid_z_vox'],
            ], dtype=np.float64)
            jitter = np.asarray([
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-8, 9),
                jitter_rng.integers(-3, 4),
            ])
            shape = np.asarray(
                pilot_mask_records[subject_id]['shape'], dtype=np.int64
            )
            center = np.clip(np.rint(center).astype(np.int64) + jitter, 0, shape - 1)
            positives.append({
                'epoch': epoch,
                'sample_type': 'positive',
                'subject_id': subject_id,
                'center_native': tuple(int(value) for value in center),
                'atomic_component_id': int(
                    component['atomic_component_id']
                ),
                'component_tier': tier,
                'component_volume_ml': float(
                    component['component_volume_ml']
                ),
                'r8_tier_cycle_index': int(
                    component['r8_tier_cycle_index']
                ),
                'r8_tier_cycle_offset': int(
                    component['r8_tier_cycle_offset']
                ),
            })
    if len(positives) != R8_COMPONENT_POSITIVE_PATCHES:
        raise RuntimeError(
            f'R8 epoch {epoch} has {len(positives)} positive patches.'
        )
    inherited = pilot_build_epoch_schedule(epoch)
    negatives = [
        dict(item) for item in inherited
        if str(item['sample_type']) != 'positive'
    ]
    if sum(str(item['sample_type']) == 'topology_negative' for item in negatives) != 28:
        raise RuntimeError('R8 topology-negative count changed.')
    if sum(str(item['sample_type']) == 'brain_negative' for item in negatives) != 28:
        raise RuntimeError('R8 brain-negative count changed.')
    for item in negatives:
        item['epoch'] = epoch
        item['r8_tier_cycle_index'] = -1
        item['r8_tier_cycle_offset'] = -1
    schedule = positives + negatives
    rng = np.random.default_rng(R8_SEED + epoch * 1_299_721)
    rng.shuffle(schedule)
    if len(schedule) != 168:
        raise RuntimeError(f'R8 epoch {epoch} has {len(schedule)} samples.')
    return schedule


def r8_schedule_audit():
    schedules = []
    repeated = []
    for epoch in range(1, R8_SCHEDULE_EPOCHS + 1):
        first = r8_build_epoch_schedule(epoch)
        second = r8_build_epoch_schedule(epoch)
        if first != second:
            raise RuntimeError(f'R8 epoch {epoch} schedule is not deterministic.')
        schedules.extend(first)
        repeated.extend(second)
    frame = pd.DataFrame(schedules)
    positive = frame[frame['sample_type'].astype(str) == 'positive'].copy()
    positive['component_key'] = (
        positive['subject_id'].astype(str) + ':'
        + positive['atomic_component_id'].astype(int).astype(str)
    )
    expected_keys = set(
        component_df[
            component_df['component_tier'].astype(str).isin(R8_TIER_QUOTAS)
        ].assign(component_key=lambda value:
            value['subject_id'].astype(str) + ':'
            + value['atomic_component_id'].astype(int).astype(str)
        )['component_key']
    )
    observed_keys = set(positive['component_key'])
    rows = []
    for tier, quota in R8_TIER_QUOTAS.items():
        tier_positive = positive[
            positive['component_tier'].astype(str) == tier
        ]
        counts = tier_positive.groupby('component_key').size()
        records = R8_COMPONENT_RECORDS_BY_TIER[tier]
        completed_cycles_are_unique = True
        for cycle_index, cycle in tier_positive.groupby('r8_tier_cycle_index'):
            keys = cycle['component_key'].tolist()
            if len(cycle) == len(records) and len(keys) != len(set(keys)):
                completed_cycles_are_unique = False
        rows.append({
            'component_tier': tier,
            'tier_components': len(records),
            'quota_per_epoch': int(quota),
            'total_selections_40_epochs': int(len(tier_positive)),
            'observed_components': int(counts.size),
            'never_selected_components': int(len(records) - counts.size),
            'minimum_selections': int(counts.min()),
            'median_selections': float(counts.median()),
            'mean_selections': float(counts.mean()),
            'maximum_selections': int(counts.max()),
            'selection_gini': r7d_gini(counts.to_numpy()),
            'maximum_minus_minimum_selections': int(
                counts.max() - counts.min()
            ),
            'complete_cycles_have_no_duplicates': bool(
                completed_cycles_are_unique
            ),
        })
    summary = pd.DataFrame(rows)
    per_epoch = positive.groupby('epoch').agg(
        positive_patches=('component_key', 'size'),
        unique_components=('component_key', 'nunique'),
        unique_subjects=('subject_id', 'nunique'),
    ).reset_index()
    summary.attrs['all_eligible_components_covered'] = (
        observed_keys == expected_keys
    )
    summary.attrs['minimum_unique_positive_subjects_per_epoch'] = int(
        per_epoch['unique_subjects'].min()
    )
    atomic_write_csv(frame, R8_SCHEDULE_PATH)
    atomic_write_csv(summary, R8_SCHEDULE_SUMMARY_PATH)
    return frame, summary, per_epoch, expected_keys, observed_keys


def r8_native_component_targets(row, local_origin, component_id):
    subject_id = str(row['subject_id'])
    binary, soft, support, spacing = r7d_component_support(row)
    labels, count = ndimage.label(
        support, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    expected = len(component_records.get(subject_id, []))
    if int(count) != int(expected):
        raise RuntimeError(
            f'{subject_id}: R8 reconstructed {count} components; expected {expected}.'
        )
    component_id = int(component_id)
    if not 1 <= component_id <= count:
        raise RuntimeError(f'{subject_id}: invalid component {component_id}.')
    selected = labels == component_id
    selected_canvas, start_a = symmetric_pad(
        selected.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    support_canvas, start_b = symmetric_pad(
        support.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(support, dtype=np.uint8),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if not (start_a == start_b == start_c):
        raise RuntimeError(f'{subject_id}: R8 target padding disagreement.')
    exact = crop_at(
        selected_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    full = crop_at(
        support_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    domain = crop_at(
        domain_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    if not bool(exact.any()):
        raise RuntimeError(f'{subject_id}: R8 native exact component is empty.')
    distance = ndimage.distance_transform_edt(~exact, sampling=spacing)
    negative = (
        (distance >= R8_NATIVE_INNER_RADIUS_MM)
        & (distance <= R8_NATIVE_OUTER_RADIUS_MM)
        & domain & ~full
    )
    if not bool(negative.any()):
        raise RuntimeError(f'{subject_id}: R8 native annulus is empty.')
    retained_fraction = float(exact.sum() / max(selected.sum(), 1))
    return {
        'native_component_target': exact[None],
        'native_component_negative_annulus': negative[None],
        'spacing': spacing,
        'component_native_voxels': int(selected.sum()),
        'component_patch_voxels': int(exact.sum()),
        'component_retained_fraction': retained_fraction,
        'negative_annulus_voxels': int(negative.sum()),
        'binary': binary,
        'soft': soft,
    }


def r8_native_target_probe(schedule_frame):
    probes = []
    positive = schedule_frame[
        schedule_frame['sample_type'].astype(str) == 'positive'
    ]
    for tier in R8_TIER_QUOTAS:
        candidates = positive[
            positive['component_tier'].astype(str) == tier
        ].head(8)
        for item in candidates.to_dict('records'):
            subject_id = str(item['subject_id'])
            row = fit_row_by_subject[subject_id]
            local_origin = pilot_origin_from_center(
                item['center_native'], pilot_mask_records[subject_id]['shape']
            )
            target = r8_native_component_targets(
                row, local_origin, item['atomic_component_id']
            )
            probes.append({
                'subject_id': subject_id,
                'component_tier': tier,
                'atomic_component_id': int(item['atomic_component_id']),
                'component_native_voxels': target['component_native_voxels'],
                'component_patch_voxels': target['component_patch_voxels'],
                'component_retained_fraction': target[
                    'component_retained_fraction'
                ],
                'negative_annulus_voxels': target[
                    'negative_annulus_voxels'
                ],
                'exact_and_annulus_disjoint': not bool(np.any(
                    target['native_component_target']
                    & target['native_component_negative_annulus']
                )),
                'target_arrays_read_split': 'fold0_fit',
                'locked_validation_target_read': False,
                'test_target_read': False,
            })
    frame = pd.DataFrame(probes)
    atomic_write_csv(frame, R8_NATIVE_TARGET_AUDIT_PATH)
    return frame


class NativeSatelliteProposalHead(nn.Module):
    def __init__(self, local_channels=32, hidden=R8_SATELLITE_FEATURE_CHANNELS):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(local_channels + 2, hidden, 1, bias=False),
            nn.GroupNorm(group_count(hidden), hidden), nn.GELU(),
        )
        self.refine = ResidualBlock3D(hidden, hidden, dropout=0.05)
        self.output = nn.Conv3d(hidden, 1, 1)

    def forward(self, local_feature, coarse_proposal, arterial_local):
        feature = self.project(torch.cat([
            local_feature, torch.sigmoid(coarse_proposal), arterial_local,
        ], dim=1))
        return self.output(self.refine(feature))


class VascuFateNetR8(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.base = VascuFateNetV22(schema)
        self.satellite_head = NativeSatelliteProposalHead()
        self.satellite_gate = ConfidenceGate(
            confidence_dim=5, output_dim=1,
            bound=R8_SATELLITE_GATE_BOUND, hidden_dim=32,
        )
        self._decoder_feature = None
        self._decoder_hook = self.base.local_model.decoder1.register_forward_hook(
            self._capture_decoder_feature
        )
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def _capture_decoder_feature(self, module, inputs, output):
        self._decoder_feature = output

    def load_r5_base_state(self, state):
        self.base.load_state_dict(state, strict=True)

    def activate_satellite_gate_for_gradient_test(self, fraction=0.5):
        self.satellite_gate.activate_for_gradient_test(fraction)

    def reset_satellite_gate_to_zero(self):
        with torch.no_grad():
            self.satellite_gate.network[-1].weight.zero_()
            self.satellite_gate.network[-1].bias.zero_()

    def forward(self, **kwargs):
        self._decoder_feature = None
        output = self.base(**kwargs)
        if self._decoder_feature is None:
            raise RuntimeError('R8 decoder feature hook did not fire.')
        arterial_local = sample_global_to_local(
            output['arterial_global_map'][:, None],
            kwargs['local_origin_in_canvas'],
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        if bool(kwargs.get('force_global_dropout', False)):
            coarse = torch.zeros_like(output['global_proposal_local'])
            arterial_local = torch.zeros_like(arterial_local)
        else:
            coarse = output['global_proposal_local']
        satellite_logits = self.satellite_head(
            self._decoder_feature, coarse, arterial_local
        )
        confidence = torch.cat([
            kwargs['confidence_features'],
            torch.sigmoid(output['lesion_presence_logit'])[:, None],
        ], dim=-1)
        gate = self.satellite_gate(confidence)
        base_logits = output['logits']
        fused_logits = base_logits + gate[..., None, None, None] * satellite_logits
        self._decoder_feature = None
        return {
            **output,
            'base_logits': base_logits,
            'satellite_logits': satellite_logits,
            'satellite_gate': gate,
            'logits': fused_logits,
        }


def r8_satellite_component_loss(
    logits, exact_target, negative_annulus, component_valid,
):
    logits = logits.float()
    exact_target = exact_target.bool()
    negative_annulus = negative_annulus.bool()
    valid = component_valid.bool().reshape(-1)
    losses = []
    rows = []
    for batch_index in range(logits.shape[0]):
        if not bool(valid[batch_index]):
            continue
        positive = logits[batch_index][exact_target[batch_index]]
        negative = logits[batch_index][negative_annulus[batch_index]]
        if positive.numel() == 0 or negative.numel() == 0:
            raise RuntimeError('R8 component loss received an empty region.')
        hard_count = min(
            int(negative.numel()),
            max(32, 4 * int(positive.numel())),
        )
        hard_negative = torch.topk(negative, hard_count).values
        positive_bce = F.softplus(-positive).mean()
        negative_bce = F.softplus(hard_negative).mean()
        rank = F.relu(
            1.0 - positive.mean() + hard_negative.mean()
        )
        absolute_floor = F.softplus(
            R8_ABSOLUTE_POSITIVE_LOGIT_FLOOR - positive.mean()
        )
        loss = (
            0.35 * positive_bce + 0.30 * negative_bce
            + 0.20 * rank + 0.15 * absolute_floor
        )
        losses.append(loss)
        rows.append({
            'positive_voxels': int(positive.numel()),
            'negative_voxels': int(negative.numel()),
            'hard_negative_voxels': int(hard_count),
            'positive_logit_mean': float(positive.mean().detach().cpu()),
            'hard_negative_logit_mean': float(
                hard_negative.mean().detach().cpu()
            ),
            'relative_logit_margin': float(
                (positive.mean() - hard_negative.mean()).detach().cpu()
            ),
            'positive_bce': float(positive_bce.detach().cpu()),
            'hard_negative_bce': float(negative_bce.detach().cpu()),
            'rank_loss': float(rank.detach().cpu()),
            'absolute_positive_floor_loss': float(
                absolute_floor.detach().cpu()
            ),
            'total_satellite_loss': float(loss.detach().cpu()),
        })
    if not losses:
        return logits.sum() * 0.0, []
    return torch.stack(losses).mean(), rows


def r8_model_input_batch(sample, device):
    return {
        key: torch.from_numpy(np.asarray(sample[key])[None]).to(device)
        for key in MODEL_INPUT_KEYS_V22
    }


def r8_gradient_norm(module):
    values = [
        parameter.grad.detach().float().norm().cpu().item()
        for parameter in module.parameters()
        if parameter.grad is not None
    ]
    if not values:
        return 0.0, False, True
    finite = bool(np.isfinite(values).all())
    return float(np.linalg.norm(values)), True, finite


def run_r8_dual_resolution_preflight():
    if r8_existing_preflight_complete():
        gate = load_json(R8_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Total parameters: {gate["total_parameters"]:,}')
        print(f'Preflight gate:   {R8_GATE_PATH}')
        print('No source checkpoint or preflight artifact was rewritten.')
        return

    r7_failure_gate = load_json(R7_FAILURE_GATE_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r7_result = load_json(PILOT_RESULT_GATE_PATH)
    r5_recomputed = dict(
        r7_result['r5_recomputed_under_r7_metric_contract']
    )
    r5_mean_dice = float(r5_recomputed['positive_mean_dice'])
    r5_global_dice = float(r5_recomputed['positive_global_dice'])
    r8_mean_dice_eligibility_floor = r5_mean_dice - 0.005
    r8_global_dice_eligibility_floor = r5_global_dice - 0.010
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r7_failure_sha = file_sha256(R7_FAILURE_GATE_PATH)
    upstream_checks = {
        'r7_failure_diagnostic_complete': (
            r7_failure_gate.get('diagnostic_complete') is True
        ),
        'r7_failure_diagnostic_passed': (
            r7_failure_gate.get('diagnostic_passed') is True
        ),
        'r8_preflight_allowed_by_diagnostic': (
            r7_failure_gate.get('r8_preflight_implementation_allowed') is True
        ),
        'r8_training_was_locked_by_diagnostic': (
            r7_failure_gate.get('r8_training_authorized') is False
        ),
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r5_recomputed_baseline_is_finite': bool(np.isfinite([
            r5_mean_dice, r5_global_dice,
        ]).all()),
        'r5_checkpoint_hash_matches': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'fold0_fit_count_is_82': len(fold0_fit_df) == 82,
        'fold0_heldout_targets_not_used': len(fold0_heldout_df) == 20,
        'locked_validation_targets_not_used': len(locked_validation_df) == 23,
        'sealed_test_targets_not_used': len(sealed_test_df) == 23,
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError('R8 upstream checks failed: ' + ', '.join(failed))

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT STARTING')
    print('=' * 80)
    print('Warm-start source:                  completed r5 best checkpoint')
    print('R7 checkpoint used for weights:     False')
    print('Target access:                      fold-0 fit probes only')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer / training steps:         0 / 0')

    schedule, schedule_summary, per_epoch, expected_keys, observed_keys = (
        r8_schedule_audit()
    )
    target_probe = r8_native_target_probe(schedule)

    positive_schedule = schedule[
        schedule['sample_type'].astype(str) == 'positive'
    ]
    smoke_candidates = positive_schedule[
        positive_schedule['component_tier'].astype(str).isin(['micro', 'tiny'])
    ]
    smoke_entry = None
    for item in smoke_candidates.to_dict('records'):
        if as_bool(fit_row_by_subject[str(item['subject_id'])][
            'trial8_atlas_enabled'
        ]):
            smoke_entry = item
            break
    if smoke_entry is None:
        raise RuntimeError('R8 found no atlas-enabled micro/tiny smoke case.')
    subject_id = str(smoke_entry['subject_id'])
    row = fit_row_by_subject[subject_id]
    local_origin = pilot_origin_from_center(
        smoke_entry['center_native'], pilot_mask_records[subject_id]['shape']
    )
    dataset = Trial8V22Fold0Dataset()
    dataset.epoch = int(smoke_entry['epoch'])
    dataset.schedule = [smoke_entry]
    sample = dataset[0]
    native_target = r8_native_component_targets(
        row, local_origin, smoke_entry['atomic_component_id']
    )

    if not torch.cuda.is_available():
        raise RuntimeError('R8 preflight requires an L4-class CUDA runtime.')
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < 20.0:
        raise RuntimeError(
            f'R8 preflight expected >=20 GB GPU, observed {gpu_capacity_gb:.2f} GB.'
        )
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'r5_checkpoint_pilot_version_matches': (
            checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'r5_checkpoint_code_revision_matches': (
            checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
        ),
        'r5_checkpoint_parameter_count_matches': int(
            checkpoint.get('parameter_count', -1)
        ) == EXPECTED_V22_PARAMETERS,
        'r5_checkpoint_contains_ema_state': 'ema_model_state' in checkpoint,
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 warm start: ' + ', '.join(failed))
    batch = r8_model_input_batch(sample, device)
    force_flags = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }

    reference = VascuFateNetV22(pilot_schema).to(device).eval()
    reference.load_state_dict(checkpoint['ema_model_state'], strict=True)
    for parameter in reference.parameters():
        parameter.requires_grad_(False)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        reference_output = reference(**batch, **force_flags)
    reference_cpu = {
        key: value.detach().float().cpu()
        for key, value in reference_output.items()
        if torch.is_tensor(value)
    }
    del reference, reference_output
    gc.collect()
    torch.cuda.empty_cache()

    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(checkpoint['ema_model_state'])
    del checkpoint
    model.eval()
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    base_parameters = sum(parameter.numel() for parameter in model.base.parameters())
    satellite_head_parameters = sum(
        parameter.numel() for parameter in model.satellite_head.parameters()
    )
    satellite_gate_parameters = sum(
        parameter.numel() for parameter in model.satellite_gate.parameters()
    )
    new_parameters = satellite_head_parameters + satellite_gate_parameters
    parameter_frame = pd.DataFrame([
        {'component': 'preserved_r5_v2_2_base', 'parameters': base_parameters},
        {'component': 'native_satellite_head', 'parameters': satellite_head_parameters},
        {'component': 'bounded_satellite_gate', 'parameters': satellite_gate_parameters},
        {'component': 'new_r8_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
    ])
    atomic_write_csv(parameter_frame, R8_PARAMETER_PATH)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        neutral_output = model(**batch, **force_flags)
    peak_eval_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    preserved_keys = [
        'logits', 'auxiliary_half_logits', 'auxiliary_quarter_logits',
        'territory_node_logits', 'global_graph_context',
        'global_proposal_logits', 'lesion_presence_logit',
        'log_volume_prediction',
    ]
    neutrality_rows = []
    for key in preserved_keys:
        difference = float(torch.max(torch.abs(
            neutral_output[key].detach().float().cpu() - reference_cpu[key]
        )))
        neutrality_rows.append({
            'output': key,
            'maximum_absolute_difference_from_r5': difference,
            'exactly_preserved_with_zero_gate': difference <= 1e-6,
        })
    neutrality_frame = pd.DataFrame(neutrality_rows)
    atomic_write_csv(neutrality_frame, R8_NEUTRALITY_PATH)

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'base_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'satellite_gate': (1, 1),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
    }
    output_rows = []
    for key, expected in expected_shapes.items():
        observed = tuple(neutral_output[key].shape)
        output_rows.append({
            'output': key,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(neutral_output[key]).all().item()),
        })
    output_frame = pd.DataFrame(output_rows)
    atomic_write_csv(output_frame, R8_OUTPUT_PATH)

    zero_gate = float(neutral_output['satellite_gate'].abs().max().cpu())
    satellite_std = float(
        neutral_output['satellite_logits'].float().std().cpu()
    )
    model.activate_satellite_gate_for_gradient_test(0.5)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        activated_output = model(**batch, **force_flags)
    activated_difference = float(torch.max(torch.abs(
        activated_output['logits'].float()
        - activated_output['base_logits'].float()
    )).cpu())
    model.reset_satellite_gate_to_zero()
    del activated_output

    model.train()
    model.zero_grad(set_to_none=True)
    exact_target = torch.from_numpy(
        native_target['native_component_target'][None]
    ).to(device)
    negative_target = torch.from_numpy(
        native_target['native_component_negative_annulus'][None]
    ).to(device)
    component_valid = torch.ones(1, dtype=torch.bool, device=device)
    local_binary = torch.from_numpy(
        np.asarray(sample['target_binary'])[None]
    ).to(device)
    global_target = torch.from_numpy(
        np.asarray(sample['global_target'])[None]
    ).to(device)
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        gradient_output = model(**batch, **force_flags)
        main_loss = (
            0.60 * pilot_soft_dice_loss(
                gradient_output['logits'], local_binary
            ).mean()
            + 0.40 * pilot_focal_bce_loss(
                gradient_output['logits'], local_binary
            ).mean()
        )
        coarse_loss = (
            0.70 * pilot_soft_dice_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
            + 0.30 * pilot_focal_bce_loss(
                gradient_output['global_proposal_logits'], global_target
            ).mean()
        )
        satellite_loss, satellite_loss_rows = r8_satellite_component_loss(
            gradient_output['satellite_logits'], exact_target,
            negative_target, component_valid,
        )
        smoke_loss = main_loss + 0.20 * coarse_loss + 0.10 * satellite_loss
    smoke_loss.backward()
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_groups = {
        'preserved_local_segmentation_head': (
            model.base.local_model.segmentation_head
        ),
        'preserved_global_proposal_stream': model.base.global_stream,
        'native_satellite_head': model.satellite_head,
        'bounded_satellite_gate': model.satellite_gate,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_groups.items():
        norm, observed, finite = r8_gradient_norm(module)
        gradient_rows.append({
            'component': name,
            'gradient_norm': norm,
            'gradient_observed': observed,
            'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_frame = pd.DataFrame(gradient_rows)
    atomic_write_csv(gradient_frame, R8_GRADIENT_PATH)
    loss_frame = pd.DataFrame([{
        'subject_id': subject_id,
        'component_tier': str(smoke_entry['component_tier']),
        'component_id': int(smoke_entry['atomic_component_id']),
        'main_loss': float(main_loss.detach().float().cpu()),
        'coarse_whole_mask_loss': float(coarse_loss.detach().float().cpu()),
        'native_satellite_loss': float(
            satellite_loss.detach().float().cpu()
        ),
        'total_smoke_loss': float(smoke_loss.detach().float().cpu()),
        **satellite_loss_rows[0],
    }])
    atomic_write_csv(loss_frame, R8_LOSS_PATH)
    model.zero_grad(set_to_none=True)

    architecture = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'warm_start': {
            'source': 'completed_r5_best_ema',
            'checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'checkpoint_sha256': r5_sha_before,
            'r7_weights_used': False,
            'zero_gate_output_equivalence_required': True,
        },
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'native_satellite_head': {
            'input': 'native-resolution_r5_decoder_feature_plus_coarse_proposal_plus_arterial_map',
            'input_spatial_shape': list(LOCAL_PATCH_SIZE),
            'hidden_channels': R8_SATELLITE_FEATURE_CHANNELS,
            'output': 'native_resolution_satellite_logits',
            'direct_graph_to_logit_residual': False,
        },
        'fusion': {
            'type': 'zero_initialized_bounded_scalar_residual',
            'bound': R8_SATELLITE_GATE_BOUND,
            'gate_context': 'registration_confidence_plus_lesion_presence',
            'initial_final_output': 'exact_r5_logits',
        },
        'coarse_global_role': (
            'whole_brain_burden_and_broad_localization_only; '
            'no_r7_coarse_component_contrastive_loss'
        ),
        'clinical_and_arterial_role': (
            'preserved_bounded_v2_2_context; no treatment/outcome leakage'
        ),
    }
    training_policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'experiment': 'R8_fold0_pilot_only_after_separate_training_preflight',
        'component_cycle': {
            'positive_patches_per_epoch': R8_COMPONENT_POSITIVE_PATCHES,
            'tier_quotas': R8_TIER_QUOTAS,
            'rule': 'within_each_tier_every_component_once_before_repeat',
            'schedule_epochs_audited': R8_SCHEDULE_EPOCHS,
        },
        'optimization_stages': [
            {
                'epochs': '1-4',
                'trainable': ['native_satellite_head'],
                'frozen': ['r5_base', 'satellite_gate'],
                'purpose': 'learn_absolute_native_component_evidence_before_fusion',
            },
            {
                'epochs': '5-12',
                'trainable': ['native_satellite_head', 'satellite_gate'],
                'frozen': ['r5_base'],
                'purpose': 'learn_bounded_residual_without_erasing_r5_behavior',
            },
            {
                'epochs': '13+',
                'trainable': ['all'],
                'learning_rate_policy': (
                    'r5_base_2e-5_satellite_2e-4_gate_1e-4'
                ),
                'purpose': 'low_rate_joint_calibration',
            },
        ],
        'loss_contract': {
            'main_segmentation': 'preserved_r5_dice_plus_focal',
            'coarse_proposal': 'whole_mask_only',
            'native_satellite': (
                'per_component_positive_bce_plus_hard_negative_bce_plus_'
                'ranking_plus_absolute_positive_logit_floor'
            ),
            'coarse_component_contrastive': False,
            'volume_case_weights': False,
            'topk_ohem': False,
        },
        'checkpoint_eligibility': {
            'r5_recomputed_mean_dice': r5_mean_dice,
            'r5_recomputed_global_dice': r5_global_dice,
            'mean_dice_minimum': r8_mean_dice_eligibility_floor,
            'global_dice_minimum': r8_global_dice_eligibility_floor,
            'small_component_improvement_required': True,
            'deployment_metrics_only': True,
            'oracle_volume_budget_used_for_selection': False,
            'selection_order': [
                'eligibility_guards', 'mean_dice', 'small_case_dice',
                'small_component_recall', 'lesion_f1',
            ],
        },
        'fold0_heldout_targets_for_training': False,
        'locked_validation_targets_for_training': False,
        'test_targets_for_training': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(architecture, R8_ARCHITECTURE_PATH)
    atomic_write_json(training_policy, R8_TRAINING_POLICY_PATH)

    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    preflight_checks = {
        **upstream_checks,
        **checkpoint_checks,
        'tier_quotas_sum_to_112': sum(R8_TIER_QUOTAS.values()) == 112,
        'schedule_has_40_epochs': schedule['epoch'].nunique() == 40,
        'every_epoch_has_168_samples': bool(
            (schedule.groupby('epoch').size() == 168).all()
        ),
        'every_epoch_has_112_positive_samples': bool(
            (schedule[schedule['sample_type'] == 'positive']
             .groupby('epoch').size() == 112).all()
        ),
        'all_eligible_components_are_covered': observed_keys == expected_keys,
        'no_eligible_component_is_never_selected': bool(
            (schedule_summary['never_selected_components'] == 0).all()
        ),
        'component_selection_counts_differ_by_at_most_one': bool(
            (schedule_summary['maximum_minus_minimum_selections'] <= 1).all()
        ),
        'complete_tier_cycles_have_no_duplicates': bool(
            schedule_summary['complete_cycles_have_no_duplicates'].all()
        ),
        'component_cycle_gini_below_0p10': bool(
            (schedule_summary['selection_gini'] < 0.10).all()
        ),
        'native_target_probes_cover_four_tiers': (
            target_probe['component_tier'].nunique() == 4
        ),
        'all_native_target_probes_are_nonempty': bool(
            (target_probe['component_patch_voxels'] > 0).all()
        ),
        'all_native_annuli_are_nonempty': bool(
            (target_probe['negative_annulus_voxels'] > 0).all()
        ),
        'all_native_targets_and_annuli_are_disjoint': bool(
            target_probe['exact_and_annulus_disjoint'].all()
        ),
        'all_native_target_probes_retain_at_least_95_percent': bool(
            (target_probe['component_retained_fraction'] >= 0.95).all()
        ),
        'preserved_base_parameter_count_matches_r5': (
            base_parameters == EXPECTED_V22_PARAMETERS
        ),
        'new_parameter_count_within_limit': (
            0 < new_parameters <= R8_MAXIMUM_NEW_PARAMETERS
        ),
        'all_zero_gate_outputs_match_r5': bool(
            neutrality_frame['exactly_preserved_with_zero_gate'].all()
        ),
        'satellite_gate_is_exactly_zero_initially': zero_gate == 0.0,
        'satellite_logits_are_nonconstant': satellite_std > 1e-6,
        'activated_satellite_gate_changes_final_logits': (
            activated_difference > 1e-7
        ),
        'all_output_shapes_match': bool(output_frame['shape_matches'].all()),
        'all_outputs_are_finite': bool(output_frame['finite'].all()),
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss).item()),
        'satellite_loss_contains_absolute_positive_floor': (
            float(loss_frame['absolute_positive_floor_loss'].iloc[0]) > 0.0
        ),
        **gradient_checks,
        'peak_eval_memory_below_20gb': peak_eval_memory_gb < 20.0,
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'r7_checkpoint_not_loaded_for_weights': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'r8_pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in preflight_checks.items()
    ])
    atomic_write_csv(checks_frame, R8_CHECKS_PATH)
    preflight_passed = all(preflight_checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R8_OUTPUT_ARTIFACTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8_PREFLIGHT_VERSION,
        'code_revision': R8_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_checkpoint_used_for_weights': False,
        'r7_failure_diagnostic_path': str(R7_FAILURE_GATE_PATH),
        'r7_failure_diagnostic_sha256': r7_failure_sha,
        'preserved_base_parameters': base_parameters,
        'new_parameters': new_parameters,
        'total_parameters': total_parameters,
        'gpu': gpu_name,
        'gpu_capacity_gb': gpu_capacity_gb,
        'peak_eval_memory_gb': peak_eval_memory_gb,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'schedule_components': len(expected_keys),
        'minimum_unique_positive_subjects_per_epoch': int(
            per_epoch['unique_subjects'].min()
        ),
        'native_target_probes': len(target_probe),
        'optimizer_steps': 0,
        'training_performed': False,
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'r8_pilot_implementation_allowed': bool(preflight_passed),
        'r8_pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'checks': preflight_checks,
        'artifact_sha256': artifact_hashes,
        'architecture_path': str(R8_ARCHITECTURE_PATH),
        'training_policy_path': str(R8_TRAINING_POLICY_PATH),
    }
    atomic_write_json(gate, R8_GATE_PATH)
    del model, neutral_output, gradient_output, batch
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in preflight_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 DUAL-RESOLUTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR8 component-cycle summary:')
    display(schedule_summary)
    print('\nR8 parameter count:')
    display(parameter_frame)
    print('\nR8 output shapes:')
    display(output_frame)
    print('\nR8 gradient smoke:')
    display(gradient_frame)
    print('\nR8 preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Warm-start checkpoint:              r5 epoch {r5_result["best_epoch"]}')
    print('R7 weights used:                    False')
    print(f'Preserved base parameters:          {base_parameters:,}')
    print(f'New satellite parameters:           {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Eligible components cycled:         {len(expected_keys)}')
    print(f'Minimum unique subjects per epoch:  {int(per_epoch["unique_subjects"].min())}')
    print(f'Peak gradient memory GB:            {peak_gradient_memory_gb:.3f}')
    print('Coarse component contrastive loss:  disabled')
    print('Native absolute calibration loss:   enabled')
    print('Initial final logits equal r5:      True')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Training performed:                 False')
    print('R8 pilot implementation allowed:    True')
    print('R8 pilot training authorized:       False')
    print('Full five-fold training authorized: False')
    print(f'Architecture:                       {R8_ARCHITECTURE_PATH}')
    print(f'Training policy:                    {R8_TRAINING_POLICY_PATH}')
    print(f'Preflight gate:                     {R8_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 R8 pilot loader/loss/training '
        'preflight from this frozen contract; do not train yet.'
    )


run_r8_dual_resolution_preflight()












R8P_PREFLIGHT_VERSION = 'v1_stagewise_native_satellite_pilot_contract'
R8P_CODE_REVISION = 'r1_negative_hard_background_pareto_resume_guard'
R8P_PILOT_VERSION = 'v2_3_fold0_dual_resolution_satellite_r8'
R8P_PILOT_CODE_REVISION = 'r1_r5_warmstart_stagewise_native_satellite'
R8P_SEED = 20261008
R8P_MAX_EPOCHS = 40
R8P_MINIMUM_EPOCHS = 20
R8P_EARLY_STOPPING_PATIENCE = 10
R8P_EARLY_STOPPING_MINIMUM_DELTA = 0.002
R8P_ACCUMULATION_STEPS = 4
R8P_HEAD_LR = 2.0e-4
R8P_GATE_LR = 1.0e-4
R8P_BASE_LR = 2.0e-5
R8P_MINIMUM_LR_FRACTION = 0.10
R8P_WEIGHT_DECAY = 1.0e-4
R8P_GRADIENT_CLIP_NORM = 2.0
R8P_SATELLITE_LOSS_COEFFICIENT = 0.10
R8P_NEGATIVE_BACKGROUND_WEIGHT = 0.25
R8P_NEGATIVE_BACKGROUND_TOPK = 4096
R8P_VALIDATION_OVERLAP = 0.50
R8P_LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
R8P_PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
R8P_EXPECTED_PARAMETERS = 16_885_385

R8P_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / R8_MODEL_VERSION / R8P_PREFLIGHT_VERSION
)
R8P_ROOT.mkdir(parents=True, exist_ok=True)
R8P_POLICY_PATH = R8P_ROOT / 'r8_fold0_pilot_policy.json'
R8P_RESUME_PATH = R8P_ROOT / 'r8_resume_contract.json'
R8P_LOADER_AUDIT_PATH = R8P_ROOT / 'r8_loader_case_audit.csv'
R8P_BATCH_AUDIT_PATH = R8P_ROOT / 'r8_batch_tensor_audit.csv'
R8P_STAGE_AUDIT_PATH = R8P_ROOT / 'r8_stage_trainability_audit.csv'
R8P_LR_AUDIT_PATH = R8P_ROOT / 'r8_learning_rate_audit.csv'
R8P_OPTIMIZER_AUDIT_PATH = R8P_ROOT / 'r8_optimizer_group_audit.csv'
R8P_LOSS_AUDIT_PATH = R8P_ROOT / 'r8_loss_smoke_audit.csv'
R8P_GRADIENT_AUDIT_PATH = R8P_ROOT / 'r8_stage_gradient_audit.csv'
R8P_CACHE_AUDIT_PATH = R8P_ROOT / 'r8_cached_inference_audit.csv'
R8P_SELECTION_AUDIT_PATH = R8P_ROOT / 'r8_checkpoint_policy_audit.csv'
R8P_CHECKS_PATH = R8P_ROOT / 'r8_fold0_pilot_preflight_checks.csv'
R8P_GATE_PATH = R8P_ROOT / 'r8_fold0_pilot_preflight_gate.json'

R8P_OUTPUT_ARTIFACTS = [
    R8P_POLICY_PATH, R8P_RESUME_PATH, R8P_LOADER_AUDIT_PATH,
    R8P_BATCH_AUDIT_PATH, R8P_STAGE_AUDIT_PATH, R8P_LR_AUDIT_PATH,
    R8P_OPTIMIZER_AUDIT_PATH, R8P_LOSS_AUDIT_PATH,
    R8P_GRADIENT_AUDIT_PATH, R8P_CACHE_AUDIT_PATH,
    R8P_SELECTION_AUDIT_PATH, R8P_CHECKS_PATH,
]

R8P_PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / R8_MODEL_VERSION / R8P_PILOT_VERSION
)
R8P_PILOT_CONFIG_PATH = R8P_PILOT_ROOT / 'pilot_config.json'
R8P_PILOT_START_GATE_PATH = R8P_PILOT_ROOT / 'pilot_start_gate.json'
R8P_LAST_CHECKPOINT_PATH = R8P_PILOT_ROOT / 'last_checkpoint.pt'
R8P_BEST_CHECKPOINT_PATH = R8P_PILOT_ROOT / 'best_checkpoint.pt'
R8P_HISTORY_PATH = R8P_PILOT_ROOT / 'training_history.csv'
R8P_RESULT_GATE_PATH = R8P_PILOT_ROOT / 'pilot_result_gate.json'


def r8p_existing_preflight_complete():
    if not R8P_GATE_PATH.exists():
        return False
    gate = load_json(R8P_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('preflight_version') == R8P_PREFLIGHT_VERSION
        and gate.get('code_revision') == R8P_CODE_REVISION
        and gate.get('r8_architecture_gate_sha256') == file_sha256(R8_GATE_PATH)
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R8P_OUTPUT_ARTIFACTS
    )


def r8p_parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in sorted(model.named_parameters()):
        digest.update(name.encode('utf-8'))
        digest.update(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        )
    return digest.hexdigest()


def r8p_stage(epoch):
    epoch = int(epoch)
    if not 1 <= epoch <= R8P_MAX_EPOCHS:
        raise ValueError(f'R8 pilot epoch {epoch} is outside 1..40.')
    if epoch <= 4:
        return 'satellite_head_only'
    if epoch <= 12:
        return 'satellite_head_and_gate'
    return 'low_rate_joint_calibration'


def r8p_branch_lr(epoch, start_epoch, peak_lr):
    epoch = int(epoch)
    start_epoch = int(start_epoch)
    if epoch < start_epoch:
        return 0.0
    if epoch <= start_epoch + 1:
        return float(peak_lr * (epoch - start_epoch + 1) / 2.0)
    minimum = peak_lr * R8P_MINIMUM_LR_FRACTION
    denominator = max(R8P_MAX_EPOCHS - (start_epoch + 1), 1)
    progress = np.clip(
        (epoch - (start_epoch + 1)) / denominator, 0.0, 1.0
    )
    cosine = 0.5 * (1.0 + math.cos(math.pi * float(progress)))
    return float(minimum + (peak_lr - minimum) * cosine)


def r8p_learning_rates(epoch):
    return {
        'r5_base': r8p_branch_lr(epoch, 13, R8P_BASE_LR),
        'native_satellite_head': r8p_branch_lr(epoch, 1, R8P_HEAD_LR),
        'bounded_satellite_gate': r8p_branch_lr(epoch, 5, R8P_GATE_LR),
    }


def r8p_set_requires_grad(module, value):
    for parameter in module.parameters():
        parameter.requires_grad_(bool(value))


def r8p_apply_stage(model, epoch, optimizer=None):
    stage = r8p_stage(epoch)
    model.train()
    base_trainable = stage == 'low_rate_joint_calibration'
    gate_trainable = stage != 'satellite_head_only'
    r8p_set_requires_grad(model.base, base_trainable)
    r8p_set_requires_grad(model.satellite_head, True)
    r8p_set_requires_grad(model.satellite_gate, gate_trainable)
    if not base_trainable:
        model.base.eval()
    if not gate_trainable:
        model.satellite_gate.eval()
    learning_rates = r8p_learning_rates(epoch)
    if optimizer is not None:
        observed_names = set()
        for group in optimizer.param_groups:
            name = str(group['group_name'])
            observed_names.add(name)
            group['lr'] = float(learning_rates[name])
        if observed_names != set(learning_rates):
            raise RuntimeError('R8 optimizer groups do not match LR policy.')
    return stage, learning_rates


def r8p_optimizer_groups(model):
    groups = [
        {
            'group_name': 'r5_base',
            'params': list(model.base.parameters()),
            'lr': 0.0,
            'weight_decay': R8P_WEIGHT_DECAY,
        },
        {
            'group_name': 'native_satellite_head',
            'params': list(model.satellite_head.parameters()),
            'lr': R8P_HEAD_LR / 2.0,
            'weight_decay': R8P_WEIGHT_DECAY,
        },
        {
            'group_name': 'bounded_satellite_gate',
            'params': list(model.satellite_gate.parameters()),
            'lr': 0.0,
            'weight_decay': 0.0,
        },
    ]
    identifiers = [
        id(parameter) for group in groups for parameter in group['params']
    ]
    if len(identifiers) != len(set(identifiers)):
        raise RuntimeError('R8 optimizer parameter groups overlap.')
    model_identifiers = {id(parameter) for parameter in model.parameters()}
    if set(identifiers) != model_identifiers:
        raise RuntimeError('R8 optimizer groups do not cover the model exactly.')
    return groups


class Trial8R8Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = r8_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = dict(self.schedule[int(index)])
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        positive = str(entry['sample_type']) == 'positive'
        if positive:
            native = r8_native_component_targets(
                row, local_origin, entry['atomic_component_id']
            )
            if not bool((targets['target_soft'] > 0).any()):
                raise RuntimeError(
                    f'{subject_id}: R8 positive patch lost soft support.'
                )
        else:
            if bool((targets['target_soft'] > 0).any()):
                raise RuntimeError(
                    f'{subject_id}: R8 negative patch contains target.'
                )
            native = {
                'native_component_target': np.zeros(
                    (1, *LOCAL_PATCH_SIZE), dtype=bool
                ),
                'native_component_negative_annulus': np.zeros(
                    (1, *LOCAL_PATCH_SIZE), dtype=bool
                ),
                'component_patch_voxels': 0,
                'negative_annulus_voxels': 0,
                'component_retained_fraction': 1.0,
            }
        rng = np.random.default_rng(
            R8P_SEED + self.epoch * 10_000_019 + int(index) * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'native_component_target': native['native_component_target'],
            'native_component_negative_annulus': native[
                'native_component_negative_annulus'
            ],
            'native_component_valid': np.asarray(positive, dtype=bool),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
            'component_patch_voxels': np.asarray(
                native['component_patch_voxels'], dtype=np.int64
            ),
            'negative_annulus_voxels': np.asarray(
                native['negative_annulus_voxels'], dtype=np.int64
            ),
            'component_retained_fraction': np.asarray(
                native['component_retained_fraction'], dtype=np.float32
            ),
        })
        return sample


def r8p_negative_background_loss(logits, component_valid):
    valid = component_valid.bool().reshape(-1)
    losses = []
    selected_counts = []
    for batch_index in range(logits.shape[0]):
        if bool(valid[batch_index]):
            continue
        flattened = logits[batch_index].float().reshape(-1)
        hard_count = min(R8P_NEGATIVE_BACKGROUND_TOPK, flattened.numel())
        hard = torch.topk(flattened, k=hard_count, largest=True).values
        losses.append(F.softplus(hard).mean())
        selected_counts.append(hard_count)
    if not losses:
        return logits.float().sum() * 0.0, 0
    return torch.stack(losses).mean(), int(sum(selected_counts))


def r8p_whole_proposal_loss(logits, target):
    dice = pilot_soft_dice_loss(logits, target).mean()
    focal = pilot_focal_bce_loss(logits, target).mean()
    return 0.65 * dice + 0.35 * focal, dice, focal


def r8p_compute_training_loss(
    model, cpu_batch, device, epoch, dropout_flags_override=None,
):
    batch = pilot_to_device(cpu_batch, device)
    if dropout_flags_override is None:
        rng = np.random.default_rng(R8P_SEED + int(epoch) * 1_000_003)
        flags = pilot_dropout_flags(rng)
    else:
        flags = {name: bool(value) for name, value in dropout_flags_override.items()}
    required_flags = {
        'force_atlas_dropout', 'force_vascular_dropout',
        'force_occlusion_dropout', 'force_clinical_dropout',
        'force_global_dropout', 'force_arterial_dropout',
    }
    if set(flags) != required_flags:
        raise RuntimeError('R8 dropout flags do not match the frozen schema.')
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool()
        & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    proposal_loss, proposal_dice, proposal_focal = r8p_whole_proposal_loss(
        output['global_proposal_logits'], batch['global_target'].float()
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].float().sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], batch['global_target'] > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'],
        global_territory_target, arterial_valid,
    )
    satellite_positive, satellite_rows = r8_satellite_component_loss(
        output['satellite_logits'], batch['native_component_target'],
        batch['native_component_negative_annulus'],
        batch['native_component_valid'],
    )
    satellite_negative, hard_background_voxels = (
        r8p_negative_background_loss(
            output['satellite_logits'], batch['native_component_valid']
        )
    )
    satellite_objective = (
        satellite_positive
        + R8P_NEGATIVE_BACKGROUND_WEIGHT * satellite_negative
    )
    preserved_objective = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    total = (
        preserved_objective
        + R8P_SATELLITE_LOSS_COEFFICIENT * satellite_objective
    )
    parts = {
        'total_loss': total,
        'preserved_objective': preserved_objective,
        'main_loss': main_loss,
        **main_parts,
        'half_loss': half_loss,
        'quarter_loss': quarter_loss,
        'node_loss': node_loss,
        'proposal_whole_mask_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_focal,
        'presence_loss': presence_loss,
        'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'satellite_positive_loss': satellite_positive,
        'satellite_negative_background_loss': satellite_negative,
        'satellite_objective': satellite_objective,
        'satellite_loss_coefficient': torch.as_tensor(
            R8P_SATELLITE_LOSS_COEFFICIENT,
            device=total.device, dtype=torch.float32,
        ),
        'hard_background_voxels': torch.as_tensor(
            float(hard_background_voxels),
            device=total.device, dtype=torch.float32,
        ),
        'native_components_in_batch': batch[
            'native_component_valid'
        ].float().sum(),
    }
    return total, parts, satellite_rows, output


def r8p_cached_local_from_global(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    model._decoder_feature = None
    base_output = pilot_local_from_cache(
        model.base, batch, cache,
        force_global_dropout=force_global_dropout,
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    if model._decoder_feature is None:
        raise RuntimeError('R8 cached path did not capture decoder feature.')
    coarse = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    if bool(force_global_dropout):
        coarse = torch.zeros_like(coarse)
        arterial_local = torch.zeros_like(arterial_local)
    satellite_logits = model.satellite_head(
        model._decoder_feature, coarse, arterial_local
    )
    confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    gate = model.satellite_gate(confidence)
    base_logits = base_output['logits']
    result = {
        **base_output,
        'base_logits': base_logits,
        'satellite_logits': satellite_logits,
        'satellite_gate': gate,
        'logits': base_logits + gate[..., None, None, None] * satellite_logits,
        'arterial_global_map': cache['arterial_map'],
        'global_proposal_local': coarse,
    }
    model._decoder_feature = None
    return result


def r8p_r5_small_component_recall(r5_baseline):
    rows = r5_baseline.get('component_tier_metrics', [])
    selected = [
        row for row in rows
        if str(row.get('component_tier')) in set(COMPONENT_TIERS[:4])
    ]
    denominator = sum(int(row.get('components', 0)) for row in selected)
    if denominator <= 0:
        raise RuntimeError('R5 baseline has no small-component audit rows.')
    numerator = sum(
        int(row.get('components', 0))
        * float(row.get('final_component_recall', 0.0))
        for row in selected
    )
    return float(numerator / denominator)


def r8p_r5_small_case_dice(r5_summary):
    bins = dict(r5_summary.get('lesion_bin_mean_dice', {}))
    values = [
        float(bins[name]) for name in ('tiny', 'small')
        if name in bins and np.isfinite(float(bins[name]))
    ]
    if not values:
        raise RuntimeError('R5 baseline has no finite tiny/small case Dice.')
    return float(np.mean(values))


def r8p_checkpoint_evaluation(metrics, baseline):
    required = [
        'positive_mean_dice', 'positive_global_dice',
        'small_case_mean_dice', 'small_component_recall', 'lesion_f1',
    ]
    if any(name not in metrics for name in required):
        raise RuntimeError('R8 checkpoint metrics are incomplete.')
    if not all(np.isfinite(float(metrics[name])) for name in required):
        raise RuntimeError('R8 checkpoint metrics are non-finite.')
    guards = {
        'mean_dice_floor': float(metrics['positive_mean_dice'])
        >= float(baseline['mean_dice_floor']),
        'global_dice_floor': float(metrics['positive_global_dice'])
        >= float(baseline['global_dice_floor']),
        'small_component_improved': float(metrics['small_component_recall'])
        > float(baseline['small_component_recall']),
    }
    key = (
        float(metrics['positive_mean_dice']),
        float(metrics['small_case_mean_dice']),
        float(metrics['small_component_recall']),
        float(metrics['lesion_f1']),
        float(metrics['positive_global_dice']),
    )
    return all(guards.values()), guards, key


def run_r8p_fold0_pilot_preflight():
    if r8p_existing_preflight_complete():
        gate = load_json(R8P_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 FOLD-0 PILOT PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Pilot version: {gate["pilot_version"]}')
        print(f'Preflight gate: {R8P_GATE_PATH}')
        print('No source checkpoint, model checkpoint, or artifact was rewritten.')
        return

    r8_gate = load_json(R8_GATE_PATH)
    r8_policy = load_json(R8_TRAINING_POLICY_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_baseline = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
    r5_summary = dict(r5_baseline['validation_summary'])
    r5_small_component_recall = r8p_r5_small_component_recall(r5_baseline)
    r5_small_case_dice = r8p_r5_small_case_dice(r5_summary)
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r8_sha = file_sha256(R8_GATE_PATH)
    upstream_artifacts_match = all(
        (R8_PREFLIGHT_ROOT / name).exists()
        and file_sha256(R8_PREFLIGHT_ROOT / name) == expected
        for name, expected in r8_gate.get('artifact_sha256', {}).items()
    )
    upstream_checks = {
        'r8_architecture_preflight_passed': (
            r8_gate.get('preflight_passed') is True
        ),
        'r8_all_architecture_checks_passed': (
            len(r8_gate.get('checks', {})) > 0
            and all(bool(value) for value in r8_gate['checks'].values())
        ),
        'r8_preflight_version_matches': (
            r8_gate.get('preflight_version') == R8_PREFLIGHT_VERSION
        ),
        'r8_code_revision_matches': (
            r8_gate.get('code_revision') == R8_CODE_REVISION
        ),
        'r8_artifact_hashes_match': upstream_artifacts_match,
        'r8_pilot_implementation_allowed': (
            r8_gate.get('r8_pilot_implementation_allowed') is True
        ),
        'r8_pilot_was_training_locked': (
            r8_gate.get('r8_pilot_training_authorized') is False
        ),
        'full_fivefold_was_locked': (
            r8_gate.get('full_fivefold_training_authorized') is False
        ),
        'r8_parameter_count_matches': (
            int(r8_gate.get('total_parameters', -1))
            == R8P_EXPECTED_PARAMETERS
        ),
        'r8_warm_start_is_r5': (
            str(r8_gate.get('r5_checkpoint_path'))
            == str(R5_BEST_CHECKPOINT_PATH)
        ),
        'r8_warm_start_hash_matches': (
            r8_gate.get('r5_checkpoint_sha256') == r5_sha_before
        ),
        'r7_weights_remain_excluded': (
            r8_gate.get('r7_checkpoint_used_for_weights') is False
        ),
        'r5_result_complete': r5_result.get('pilot_complete') is True,
        'r5_baseline_metrics_finite': bool(np.isfinite([
            float(r5_summary['positive_mean_dice']),
            float(r5_summary['positive_global_dice']),
            r5_small_case_dice, r5_small_component_recall,
        ]).all()),
        'fold0_partition_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_and_test_counts_match': (
            len(locked_validation_df) == 23 and len(sealed_test_df) == 23
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 PILOT PREFLIGHT UPSTREAM FAILURE. Do not continue. '
            'Failed: ' + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R8 FOLD-0 PILOT LOADER/LOSS/TRAINING PREFLIGHT STARTING')
    print('=' * 80)
    print('Warm start:                         r5 best EMA (epoch 23)')
    print('R7 weights used:                    False')
    print('Target access:                      fold-0 fit smoke cases only')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer steps:                    0')

    dataset = Trial8R8Fold0Dataset()
    selected_indices = []
    for tier in R8_TIER_QUOTAS:
        selected_indices.append(next(
            index for index, entry in enumerate(dataset.schedule)
            if str(entry['sample_type']) == 'positive'
            and str(entry['component_tier']) == tier
        ))
    selected_indices.extend([
        next(index for index, entry in enumerate(dataset.schedule)
             if str(entry['sample_type']) == 'topology_negative'),
        next(index for index, entry in enumerate(dataset.schedule)
             if str(entry['sample_type']) == 'brain_negative'),
    ])
    loader_rows = []
    positive_sample = None
    negative_sample = None
    for index in selected_indices:
        sample = dataset[index]
        float_arrays = [
            np.asarray(value) for value in sample.values()
            if isinstance(value, np.ndarray)
            and np.issubdtype(np.asarray(value).dtype, np.floating)
        ]
        row = {
            'subject_id': sample['subject_id'],
            'sample_type': sample['sample_type'],
            'component_tier': sample['component_tier'],
            'atomic_component_id': int(sample['atomic_component_id']),
            'target_soft_support_voxels': int(
                (sample['target_soft'] > 0).sum()
            ),
            'native_component_voxels': int(
                sample['native_component_target'].sum()
            ),
            'native_negative_annulus_voxels': int(
                sample['native_component_negative_annulus'].sum()
            ),
            'native_component_valid': bool(
                sample['native_component_valid']
            ),
            'component_retained_fraction': float(
                sample['component_retained_fraction']
            ),
            'all_float_arrays_finite': all(
                np.isfinite(value).all() for value in float_arrays
            ),
            'target_arrays_read_split': 'fold0_fit',
            'fold0_heldout_target_read': False,
            'locked_validation_target_read': False,
            'test_target_read': False,
        }
        loader_rows.append(row)
        if (
            positive_sample is None
            and sample['sample_type'] == 'positive'
            and sample['component_tier'] == 'micro'
        ):
            positive_sample = sample
        if negative_sample is None and sample['sample_type'] == 'brain_negative':
            negative_sample = sample
    if positive_sample is None or negative_sample is None:
        raise RuntimeError('R8 smoke samples were not resolved.')
    loader_frame = pd.DataFrame(loader_rows)
    atomic_write_csv(loader_frame, R8P_LOADER_AUDIT_PATH)

    positive_batch = torch.utils.data.default_collate([positive_sample])
    negative_batch = torch.utils.data.default_collate([negative_sample])
    expected_shapes = {
        'image': (1, 15, *LOCAL_PATCH_SIZE),
        'global_image': (1, 15, *GLOBAL_INPUT_SHAPE),
        'target_binary': (1, 1, *LOCAL_PATCH_SIZE),
        'target_soft': (1, 1, *LOCAL_PATCH_SIZE),
        'global_target': (1, 1, *GLOBAL_INPUT_SHAPE),
        'native_component_target': (1, 1, *LOCAL_PATCH_SIZE),
        'native_component_negative_annulus': (1, 1, *LOCAL_PATCH_SIZE),
        'local_origin_in_canvas': (1, 3),
    }
    batch_rows = []
    for name, expected in expected_shapes.items():
        observed = tuple(positive_batch[name].shape)
        batch_rows.append({
            'tensor': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(
                positive_batch[name].float()
            ).all().item()),
        })
    batch_frame = pd.DataFrame(batch_rows)
    atomic_write_csv(batch_frame, R8P_BATCH_AUDIT_PATH)

    random.seed(R8P_SEED)
    np.random.seed(R8P_SEED)
    torch.manual_seed(R8P_SEED)
    torch.cuda.manual_seed_all(R8P_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda':
        raise RuntimeError('Trial8 R8 fold-0 pilot preflight requires CUDA.')
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(checkpoint['ema_model_state'])
    del checkpoint
    total_parameters = sum(
        parameter.numel() for parameter in model.parameters()
    )
    parameter_digest_before = r8p_parameter_digest(model)
    optimizer_groups = r8p_optimizer_groups(model)
    r8p_apply_stage(model, 1)
    optimizer = torch.optim.AdamW(
        optimizer_groups, betas=(0.9, 0.99), eps=1e-8
    )
    optimizer_rows = []
    for group in optimizer.param_groups:
        optimizer_rows.append({
            'group_name': str(group['group_name']),
            'parameters': int(sum(
                parameter.numel() for parameter in group['params']
            )),
            'initial_lr': float(group['lr']),
            'weight_decay': float(group['weight_decay']),
            'optimizer_state_entries': int(sum(
                parameter in optimizer.state for parameter in group['params']
            )),
        })
    optimizer_frame = pd.DataFrame(optimizer_rows)
    atomic_write_csv(optimizer_frame, R8P_OPTIMIZER_AUDIT_PATH)

    stage_rows = []
    for epoch in (1, 4, 5, 12, 13, 40):
        stage, rates = r8p_apply_stage(model, epoch, optimizer)
        stage_rows.append({
            'epoch': epoch,
            'stage': stage,
            'trainable_parameters': int(sum(
                parameter.numel() for parameter in model.parameters()
                if parameter.requires_grad
            )),
            'base_trainable_parameters': int(sum(
                parameter.numel() for parameter in model.base.parameters()
                if parameter.requires_grad
            )),
            'satellite_head_trainable_parameters': int(sum(
                parameter.numel()
                for parameter in model.satellite_head.parameters()
                if parameter.requires_grad
            )),
            'satellite_gate_trainable_parameters': int(sum(
                parameter.numel()
                for parameter in model.satellite_gate.parameters()
                if parameter.requires_grad
            )),
            **{f'{name}_lr': value for name, value in rates.items()},
        })
    stage_frame = pd.DataFrame(stage_rows)
    atomic_write_csv(stage_frame, R8P_STAGE_AUDIT_PATH)
    lr_frame = stage_frame[[
        'epoch', 'stage', 'r5_base_lr', 'native_satellite_head_lr',
        'bounded_satellite_gate_lr',
    ]].copy()
    atomic_write_csv(lr_frame, R8P_LR_AUDIT_PATH)

    no_dropout = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
        'force_global_dropout': False,
        'force_arterial_dropout': False,
    }
    gradient_rows = []
    loss_rows = []
    gradient_probes = [
        (1, 'positive', positive_batch),
        (1, 'brain_negative', negative_batch),
        (5, 'positive', positive_batch),
        (13, 'positive', positive_batch),
    ]
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    for epoch, sample_role, cpu_batch in gradient_probes:
        stage, rates = r8p_apply_stage(model, epoch, optimizer)
        model.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type='cuda', dtype=torch.bfloat16, enabled=True
        ):
            loss, parts, satellite_rows, output = r8p_compute_training_loss(
                model, cpu_batch, device, epoch,
                dropout_flags_override=no_dropout,
            )
        if not bool(torch.isfinite(loss).item()):
            raise RuntimeError('R8 stage loss smoke is non-finite.')
        loss.backward()
        group_modules = {
            'r5_base': model.base,
            'native_satellite_head': model.satellite_head,
            'bounded_satellite_gate': model.satellite_gate,
        }
        for group_name, module in group_modules.items():
            norm, observed, finite = r8_gradient_norm(module)
            gradient_rows.append({
                'epoch': epoch,
                'stage': stage,
                'sample_role': sample_role,
                'parameter_group': group_name,
                'gradient_norm': norm,
                'gradient_observed': observed,
                'gradient_finite': finite,
                'group_lr': float(rates[group_name]),
            })
        loss_record = {
            'epoch': epoch,
            'stage': stage,
            'sample_role': sample_role,
        }
        for name, value in parts.items():
            loss_record[name] = float(value.detach().float().cpu())
        if satellite_rows:
            loss_record.update({
                f'native_{name}': value
                for name, value in satellite_rows[0].items()
            })
        loss_rows.append(loss_record)
        del output, loss, parts
        model.zero_grad(set_to_none=True)
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_frame = pd.DataFrame(gradient_rows)
    loss_frame = pd.DataFrame(loss_rows)
    atomic_write_csv(gradient_frame, R8P_GRADIENT_AUDIT_PATH)
    atomic_write_csv(loss_frame, R8P_LOSS_AUDIT_PATH)

    r8p_apply_stage(model, 1, optimizer)
    model.eval()
    positive_device = pilot_to_device(positive_batch, device)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        direct = model(
            **{key: positive_device[key] for key in MODEL_INPUT_KEYS_V22},
            **no_dropout,
        )
        cache = pilot_global_cache(model.base, positive_device)
        cached = r8p_cached_local_from_global(
            model, positive_device, cache,
        )
    cache_rows = []
    for name in ('logits', 'base_logits', 'satellite_logits'):
        difference = float(torch.max(torch.abs(
            direct[name].float() - cached[name].float()
        )).cpu())
        cache_rows.append({
            'output': name,
            'maximum_absolute_direct_vs_cached_difference': difference,
            'within_bfloat16_tolerance': difference <= 0.02,
        })
    initial_gate_abs_max = float(
        direct['satellite_gate'].float().abs().max().cpu()
    )
    initial_final_base_difference = float(torch.max(torch.abs(
        direct['logits'].float() - direct['base_logits'].float()
    )).cpu())
    cache_frame = pd.DataFrame(cache_rows)
    atomic_write_csv(cache_frame, R8P_CACHE_AUDIT_PATH)
    del direct, cached, cache, positive_device

    baseline = {
        'r5_mean_dice': float(r5_summary['positive_mean_dice']),
        'r5_global_dice': float(r5_summary['positive_global_dice']),
        'mean_dice_floor': float(
            r8_policy['checkpoint_eligibility']['mean_dice_minimum']
        ),
        'global_dice_floor': float(
            r8_policy['checkpoint_eligibility']['global_dice_minimum']
        ),
        'small_case_mean_dice': r5_small_case_dice,
        'small_component_recall': r5_small_component_recall,
    }
    passing_metrics = {
        'positive_mean_dice': baseline['mean_dice_floor'] + 0.010,
        'positive_global_dice': baseline['global_dice_floor'] + 0.010,
        'small_case_mean_dice': baseline['small_case_mean_dice'] + 0.010,
        'small_component_recall': baseline['small_component_recall'] + 0.010,
        'lesion_f1': 0.20,
        'oracle_volume_budgeted_recall': 0.99,
    }
    mean_failure = dict(passing_metrics)
    mean_failure['positive_mean_dice'] = baseline['mean_dice_floor'] - 0.001
    small_failure = dict(passing_metrics)
    small_failure['small_component_recall'] = baseline[
        'small_component_recall'
    ]
    oracle_variant = dict(passing_metrics)
    oracle_variant['oracle_volume_budgeted_recall'] = 0.0
    selection_tests = []
    for name, metrics in [
        ('eligible_improvement', passing_metrics),
        ('mean_dice_regression', mean_failure),
        ('no_small_component_improvement', small_failure),
        ('oracle_metric_changed_only', oracle_variant),
    ]:
        eligible, guards, key = r8p_checkpoint_evaluation(metrics, baseline)
        selection_tests.append({
            'scenario': name,
            'eligible': eligible,
            'selection_key': str(key),
            **guards,
        })
    selection_frame = pd.DataFrame(selection_tests)
    atomic_write_csv(selection_frame, R8P_SELECTION_AUDIT_PATH)

    parameter_digest_after = r8p_parameter_digest(model)
    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    expected_trainable = {
        1: 14_481,
        4: 14_481,
        5: 14_706,
        12: 14_706,
        13: R8P_EXPECTED_PARAMETERS,
        40: R8P_EXPECTED_PARAMETERS,
    }

    def gradient_entry(epoch, role, group):
        selected = gradient_frame[
            (gradient_frame['epoch'] == epoch)
            & (gradient_frame['sample_role'] == role)
            & (gradient_frame['parameter_group'] == group)
        ]
        if len(selected) != 1:
            raise RuntimeError('R8 gradient audit row is not unique.')
        return selected.iloc[0]

    policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'warm_start': {
            'source': 'r5_best_ema_epoch_23',
            'checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
            'checkpoint_sha256': r5_sha_before,
            'r7_weights_used': False,
            'parent_checkpoint_migration_allowed': False,
        },
        'epoch_schedule': {
            'maximum_epochs': R8P_MAX_EPOCHS,
            'minimum_epochs': R8P_MINIMUM_EPOCHS,
            'early_stopping_patience': R8P_EARLY_STOPPING_PATIENCE,
            'minimum_delta_for_patience_only': (
                R8P_EARLY_STOPPING_MINIMUM_DELTA
            ),
            'checkpoint_save_rule': 'every_strict_lexicographic_improvement',
            'metrics_printed_every_epoch': True,
        },
        'optimization_stages': r8_policy['optimization_stages'],
        'learning_rates': {
            'r5_base_peak': R8P_BASE_LR,
            'satellite_head_peak': R8P_HEAD_LR,
            'satellite_gate_peak': R8P_GATE_LR,
            'branch_local_two_epoch_warmup': True,
            'cosine_decay_to_fraction': R8P_MINIMUM_LR_FRACTION,
            'accumulation_steps': R8P_ACCUMULATION_STEPS,
            'gradient_clip_norm': R8P_GRADIENT_CLIP_NORM,
        },
        'loss': {
            'preserved_r5_main_and_auxiliary_objective': True,
            'coarse_proposal_whole_mask_only': True,
            'coarse_component_contrastive_disabled': True,
            'native_component_objective': (
                'positive_bce_plus_hard_negative_bce_plus_ranking_plus_'
                'absolute_positive_floor'
            ),
            'target_free_negative_objective': (
                'top4096_native_satellite_hard_background_softplus'
            ),
            'negative_background_weight': R8P_NEGATIVE_BACKGROUND_WEIGHT,
            'satellite_loss_coefficient': R8P_SATELLITE_LOSS_COEFFICIENT,
            'volume_case_weights': False,
            'topk_ohem_on_main_segmentation': False,
        },
        'validation': {
            'partition': 'fold0_heldout_development_only',
            'frequency': 'every_epoch',
            'local_threshold_grid': R8P_LOCAL_THRESHOLDS.tolist(),
            'proposal_threshold_grid': R8P_PROPOSAL_THRESHOLDS.tolist(),
            'sliding_window_overlap': R8P_VALIDATION_OVERLAP,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'per_epoch_diagnostics': [
                'mean_median_global_dice', 'lesion_precision_recall_f1',
                'lesion_volume_bin_dice', 'all_and_small_component_recall',
                'satellite_positive_and_hard_negative_logit_margin',
                'proposal_precision_ap_and_volume_ratio',
                'gate_distribution_and_thresholds',
            ],
            'every_five_epoch_diagnostics': [
                'satellite_ablation', 'global_ablation',
                'arterial_ablation', 'clinical_ablation',
                'r5_base_parameter_drift',
            ],
        },
        'checkpoint_eligibility': {
            **baseline,
            'small_component_improvement_required': True,
            'selection_order': [
                'positive_mean_dice', 'small_case_mean_dice',
                'small_component_recall', 'lesion_f1',
                'positive_global_dice',
            ],
            'oracle_metrics_used_for_selection': False,
            'thresholds_saved_with_checkpoint': True,
            'if_no_r8_epoch_is_eligible': (
                'declare_directional_failure_and_retain_r5_best_as_fallback'
            ),
        },
        'output_paths': {
            'pilot_root': str(R8P_PILOT_ROOT),
            'best_checkpoint': str(R8P_BEST_CHECKPOINT_PATH),
            'last_checkpoint': str(R8P_LAST_CHECKPOINT_PATH),
            'history': str(R8P_HISTORY_PATH),
            'result_gate': str(R8P_RESULT_GATE_PATH),
        },
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(policy, R8P_POLICY_PATH)
    policy_sha256 = file_sha256(R8P_POLICY_PATH)
    resume_contract = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'pilot_policy_sha256': policy_sha256,
        'accepted_resume_source': 'same_r8_pilot_last_checkpoint_only',
        'fresh_start_source': 'r5_best_ema_plus_deterministic_r8_initialization',
        'r7_checkpoint_migration_allowed': False,
        'r5_checkpoint_migration_after_epoch_zero_allowed': False,
        'deployment_fallback_if_no_eligible_r8_checkpoint': str(
            R5_BEST_CHECKPOINT_PATH
        ),
        'required_checkpoint_fields': [
            'pilot_version', 'code_revision', 'pilot_policy_sha256',
            'epoch', 'model_state', 'ema_model_state', 'optimizer_state',
            'rng_state', 'best_selection_key', 'best_thresholds',
            'optimizer_steps',
        ],
        'rng_restore_policy': (
            'normalize_cpu_and_cuda_rng_tensors_to_contiguous_cpu_uint8'
        ),
        'checkpoint_write_policy': (
            'atomic_last_each_epoch_and_atomic_best_on_every_strict_eligible_'
            'lexicographic_improvement'
        ),
        'retroactive_best_reconstruction': False,
        'mismatched_config_action': 'refuse_resume_without_overwrite',
    }
    atomic_write_json(resume_contract, R8P_RESUME_PATH)

    stage_counts_match = all(
        int(row['trainable_parameters']) == expected_trainable[int(row['epoch'])]
        for row in stage_rows
    )
    positive_loss_row = loss_frame[
        (loss_frame['epoch'] == 1)
        & (loss_frame['sample_role'] == 'positive')
    ].iloc[0]
    negative_loss_row = loss_frame[
        (loss_frame['epoch'] == 1)
        & (loss_frame['sample_role'] == 'brain_negative')
    ].iloc[0]
    oracle_a = selection_frame[
        selection_frame['scenario'] == 'eligible_improvement'
    ].iloc[0]
    oracle_b = selection_frame[
        selection_frame['scenario'] == 'oracle_metric_changed_only'
    ].iloc[0]
    checks = {
        **upstream_checks,
        'dataset_has_168_samples': len(dataset) == 168,
        'dataset_has_112_positive_samples': sum(
            str(item['sample_type']) == 'positive'
            for item in dataset.schedule
        ) == 112,
        'dataset_has_28_topology_negatives': sum(
            str(item['sample_type']) == 'topology_negative'
            for item in dataset.schedule
        ) == 28,
        'dataset_has_28_brain_negatives': sum(
            str(item['sample_type']) == 'brain_negative'
            for item in dataset.schedule
        ) == 28,
        'loader_smoke_covers_all_four_component_tiers': (
            set(loader_frame[
                loader_frame['sample_type'] == 'positive'
            ]['component_tier']) == set(R8_TIER_QUOTAS)
        ),
        'loader_smoke_covers_both_negative_roles': (
            {'topology_negative', 'brain_negative'}.issubset(
                set(loader_frame['sample_type'])
            )
        ),
        'positive_loader_cases_have_native_targets': bool((
            loader_frame[loader_frame['sample_type'] == 'positive'][
                'native_component_voxels'
            ] > 0
        ).all()),
        'positive_loader_cases_have_native_annuli': bool((
            loader_frame[loader_frame['sample_type'] == 'positive'][
                'native_negative_annulus_voxels'
            ] > 0
        ).all()),
        'negative_loader_cases_are_target_free': bool((
            loader_frame[loader_frame['sample_type'] != 'positive'][
                'target_soft_support_voxels'
            ] == 0
        ).all()),
        'all_loader_float_arrays_are_finite': bool(
            loader_frame['all_float_arrays_finite'].all()
        ),
        'all_batch_shapes_match': bool(batch_frame['shape_matches'].all()),
        'all_batch_tensors_are_finite': bool(batch_frame['finite'].all()),
        'total_parameter_count_matches': (
            total_parameters == R8P_EXPECTED_PARAMETERS
        ),
        'optimizer_groups_cover_model_exactly': (
            int(optimizer_frame['parameters'].sum()) == total_parameters
        ),
        'optimizer_groups_are_disjoint': True,
        'optimizer_state_is_empty': len(optimizer.state) == 0,
        'optimizer_steps_are_zero': True,
        'stage_trainable_counts_match': stage_counts_match,
        'base_lr_is_zero_before_epoch13': bool((
            lr_frame[lr_frame['epoch'] < 13]['r5_base_lr'] == 0.0
        ).all()),
        'gate_lr_is_zero_before_epoch5': bool((
            lr_frame[lr_frame['epoch'] < 5][
                'bounded_satellite_gate_lr'
            ] == 0.0
        ).all()),
        'head_lr_is_positive_from_epoch1': bool((
            lr_frame['native_satellite_head_lr'] > 0.0
        ).all()),
        'all_loss_smokes_are_finite': bool(np.isfinite(
            loss_frame[[
                'total_loss', 'preserved_objective', 'main_loss',
                'proposal_whole_mask_loss', 'presence_loss',
                'volume_loss', 'arterial_loss',
                'satellite_positive_loss',
                'satellite_negative_background_loss',
                'satellite_objective', 'satellite_loss_coefficient',
                'hard_background_voxels', 'native_components_in_batch',
            ]].to_numpy(dtype=np.float64)
        ).all()),
        'positive_native_absolute_floor_is_active': (
            float(positive_loss_row['native_absolute_positive_floor_loss'])
            > 0.0
        ),
        'negative_native_background_loss_is_active': (
            float(negative_loss_row['satellite_negative_background_loss'])
            > 0.0
            and float(negative_loss_row['hard_background_voxels'])
            == R8P_NEGATIVE_BACKGROUND_TOPK
        ),
        'coarse_component_contrastive_is_absent': (
            'local_contrastive_loss' not in loss_frame.columns
        ),
        'stage1_positive_head_gradient_present': bool(
            gradient_entry(1, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage1_negative_head_gradient_present': bool(
            gradient_entry(1, 'brain_negative', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage1_base_gradient_absent': not bool(
            gradient_entry(1, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage1_gate_gradient_absent': not bool(
            gradient_entry(1, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'stage2_head_gradient_present': bool(
            gradient_entry(5, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage2_gate_gradient_present': bool(
            gradient_entry(5, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'stage2_base_gradient_absent': not bool(
            gradient_entry(5, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage3_base_gradient_present': bool(
            gradient_entry(13, 'positive', 'r5_base')['gradient_observed']
        ),
        'stage3_head_gradient_present': bool(
            gradient_entry(13, 'positive', 'native_satellite_head')[
                'gradient_observed'
            ]
        ),
        'stage3_gate_gradient_present': bool(
            gradient_entry(13, 'positive', 'bounded_satellite_gate')[
                'gradient_observed'
            ]
        ),
        'all_observed_gradients_are_finite': bool(
            gradient_frame['gradient_finite'].all()
        ),
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'cached_validation_adapter_matches_direct_forward': bool(
            cache_frame['within_bfloat16_tolerance'].all()
        ),
        'actual_pilot_model_gate_is_zero_at_initialization': (
            initial_gate_abs_max == 0.0
        ),
        'actual_pilot_model_initial_output_equals_r5_base': (
            initial_final_base_difference == 0.0
        ),
        'eligible_checkpoint_scenario_passes': bool(
            selection_frame[
                selection_frame['scenario'] == 'eligible_improvement'
            ]['eligible'].iloc[0]
        ),
        'mean_dice_regression_is_rejected': not bool(
            selection_frame[
                selection_frame['scenario'] == 'mean_dice_regression'
            ]['eligible'].iloc[0]
        ),
        'missing_small_component_improvement_is_rejected': not bool(
            selection_frame[
                selection_frame['scenario']
                == 'no_small_component_improvement'
            ]['eligible'].iloc[0]
        ),
        'oracle_metric_cannot_change_selection': (
            bool(oracle_a['eligible']) == bool(oracle_b['eligible'])
            and str(oracle_a['selection_key']) == str(oracle_b['selection_key'])
        ),
        'checkpoint_paths_are_isolated_from_r5_and_r7': (
            R8P_BEST_CHECKPOINT_PATH not in {
                R5_BEST_CHECKPOINT_PATH, PILOT_BEST_CHECKPOINT_PATH
            }
            and R8P_LAST_CHECKPOINT_PATH != PILOT_LAST_CHECKPOINT_PATH
        ),
        'resume_contract_hashes_exact_pilot_policy': (
            resume_contract['pilot_policy_sha256']
            == file_sha256(R8P_POLICY_PATH)
        ),
        'no_r8_checkpoint_was_written': (
            not R8P_BEST_CHECKPOINT_PATH.exists()
            and not R8P_LAST_CHECKPOINT_PATH.exists()
        ),
        'model_parameters_unchanged_by_backward_smokes': (
            parameter_digest_before == parameter_digest_after
        ),
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'training_not_performed': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in checks.items()
    ])
    atomic_write_csv(checks_frame, R8P_CHECKS_PATH)
    preflight_passed = all(checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R8P_OUTPUT_ARTIFACTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R8_MODEL_VERSION,
        'preflight_version': R8P_PREFLIGHT_VERSION,
        'code_revision': R8P_CODE_REVISION,
        'pilot_version': R8P_PILOT_VERSION,
        'pilot_code_revision': R8P_PILOT_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r8_architecture_gate_path': str(R8_GATE_PATH),
        'r8_architecture_gate_sha256': r8_sha,
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_weights_used': False,
        'total_parameters': total_parameters,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'fit_targets_read': len(set(loader_frame['subject_id'])),
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': True,
        'optimizer_steps': 0,
        'training_performed': False,
        'r8_fold0_pilot_training_authorized': bool(preflight_passed),
        'full_fivefold_training_authorized': False,
        'pilot_policy_path': str(R8P_POLICY_PATH),
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'resume_contract_path': str(R8P_RESUME_PATH),
        'checks': checks,
        'artifact_sha256': artifact_hashes,
    }
    atomic_write_json(gate, R8P_GATE_PATH)

    del model, optimizer, positive_batch, negative_batch
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 FOLD-0 PILOT PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR8 loader smoke cases:')
    display(loader_frame)
    print('\nR8 staged trainability and learning rates:')
    display(stage_frame)
    print('\nR8 real-patch loss smoke:')
    display(loss_frame)
    print('\nR8 staged gradient audit:')
    display(gradient_frame)
    print('\nR8 checkpoint-policy tests:')
    display(selection_frame)
    print('\nR8 fold-0 pilot preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R8 FOLD-0 PILOT LOADER/LOSS/TRAINING PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Pilot version:                       {R8P_PILOT_VERSION}')
    print(f'Fit / held-out subjects:             {len(fold0_fit_df)} / {len(fold0_heldout_df)}')
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Total parameters:                    {total_parameters:,}')
    print(f'Peak gradient memory GB:             {peak_gradient_memory_gb:.3f}')
    print('Stage 1 negative patches train head: True')
    print('Coarse component contrastive loss:   disabled')
    print('Native absolute calibration loss:    enabled')
    print('R5 mean/global Pareto floors:         enforced')
    print('Oracle metrics select checkpoints:   False')
    print('Metrics printed:                      every epoch')
    print('Fold-0 held-out targets read:         False')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Optimizer steps:                      0')
    print('Training performed:                   False')
    print('R8 fold-0 pilot training authorized:  True')
    print('Full five-fold training authorized:   False')
    print(f'Pilot policy:                         {R8P_POLICY_PATH}')
    print(f'Resume contract:                      {R8P_RESUME_PATH}')
    print(f'Preflight gate:                       {R8P_GATE_PATH}')
    print(
        '\nNext: build and run the isolated fold-0 R8 pilot training cell '
        'from this frozen gate; do not launch five folds.'
    )


run_r8p_fold0_pilot_preflight()











R8T_IMPLEMENTATION_REVISION = 'r2_strict_r5_ema_pareto_epoch_diagnostics'
R8T_VALIDATION_ROOT = R8P_PILOT_ROOT / 'validation'
R8T_DIAGNOSTIC_ROOT = R8P_PILOT_ROOT / 'diagnostics'
R8T_STATE_PATH = R8P_PILOT_ROOT / 'pilot_state.json'
R8T_BRANCH_HISTORY_PATH = R8P_PILOT_ROOT / 'branch_reliance_history.csv'
R8T_BASE_DRIFT_HISTORY_PATH = R8P_PILOT_ROOT / 'r5_base_drift_history.csv'
R8T_SATELLITE_HISTORY_PATH = R8P_PILOT_ROOT / 'satellite_calibration_history.csv'
R8T_FINAL_SUMMARY_PATH = R8P_PILOT_ROOT / 'final_pilot_summary.json'

R8T_VALIDATION_CAPTURE = []
R8T_SMALL_TIERS = {
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
}
R8T_NO_DROPOUT = {
    'force_atlas_dropout': False,
    'force_vascular_dropout': False,
    'force_occlusion_dropout': False,
    'force_clinical_dropout': False,
    'force_global_dropout': False,
    'force_arterial_dropout': False,
}


def r8t_existing_result_complete():
    if not R8P_RESULT_GATE_PATH.exists():
        return False
    result = load_json(R8P_RESULT_GATE_PATH)
    if not (
        result.get('pilot_complete') is True
        and result.get('pilot_version') == R8P_PILOT_VERSION
        and result.get('code_revision') == R8P_PILOT_CODE_REVISION
        and result.get('implementation_revision')
        == R8T_IMPLEMENTATION_REVISION
        and result.get('pilot_policy_sha256')
        == file_sha256(R8P_POLICY_PATH)
        and result.get('preflight_gate_sha256')
        == file_sha256(R8P_GATE_PATH)
        and result.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and R8P_LAST_CHECKPOINT_PATH.exists()
        and result.get('last_checkpoint_sha256')
        == file_sha256(R8P_LAST_CHECKPOINT_PATH)
        and R8T_FINAL_SUMMARY_PATH.exists()
        and result.get('final_summary_sha256')
        == file_sha256(R8T_FINAL_SUMMARY_PATH)
        and R8P_HISTORY_PATH.exists()
    ):
        return False
    if bool(result.get('eligible_r8_checkpoint_found')):
        return (
            R8P_BEST_CHECKPOINT_PATH.exists()
            and result.get('best_checkpoint_sha256')
            == file_sha256(R8P_BEST_CHECKPOINT_PATH)
        )
    return not R8P_BEST_CHECKPOINT_PATH.exists()


def r8t_dropout_flags(epoch, batch_index):
    rng = np.random.default_rng(
        R8P_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    return pilot_dropout_flags(rng)


def r8t_ema_update(ema_model, model, optimizer_step):
    decay = min(
        EMA_MAX_DECAY,
        1.0 - 1.0 / float(int(optimizer_step) + 1),
    )
    with torch.no_grad():
        source = model.state_dict()
        destination = ema_model.state_dict()
        if source.keys() != destination.keys():
            raise RuntimeError('R8 EMA source/destination state schemas differ.')
        for name, value in destination.items():
            source_value = source[name].detach()
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source_value, alpha=1.0 - decay)
            else:
                value.copy_(source_value)


def r8t_small_component_recall(tier_rows):
    selected = [
        row for row in tier_rows
        if str(row.get('component_tier')) in R8T_SMALL_TIERS
    ]
    denominator = sum(int(row.get('components', 0)) for row in selected)
    if denominator <= 0:
        raise RuntimeError('R8 validation has no small components.')
    numerator = sum(
        int(row.get('components', 0))
        * float(row.get('final_component_recall', 0.0))
        for row in selected
    )
    return float(numerator / denominator)


def r8t_component_satellite_records(satellite_logit, target, spacing):
    target = np.asarray(target, dtype=bool)
    satellite_logit = np.asarray(satellite_logit, dtype=np.float32)
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    for component_id, component_slice in enumerate(
        ndimage.find_objects(labels), start=1
    ):
        if component_slice is None:
            continue
        component = labels[component_slice] == component_id
        values = satellite_logit[component_slice][component]
        voxels = int(component.sum())
        records.append({
            'component_id': int(component_id),
            'component_tier': pilot_component_tier(
                voxels * voxel_volume_ml, voxels
            ),
            'component_voxels': voxels,
            'satellite_logit_mean': float(values.mean()),
            'satellite_logit_maximum': float(values.max(initial=-np.inf)),
            'satellite_probability_maximum': float(
                torch.sigmoid(torch.from_numpy(values)).max().item()
            ),
        })
    if len(records) != int(count):
        raise RuntimeError('R8 satellite component diagnostic lost a component.')
    return records


@torch.no_grad()
def r8t_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model.base, base_batch,
            force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )

    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(
            native_shape[axis], LOCAL_PATCH_SIZE[axis],
            overlap=R8P_VALIDATION_OVERLAP,
        )
        for axis in range(3)
    ]
    final_sum = np.zeros(native_shape, dtype=np.float32)
    base_sum = np.zeros(native_shape, dtype=np.float32)
    satellite_logit_sum = np.zeros(native_shape, dtype=np.float32)
    gate_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][axis] + native_start[axis])
                    for axis in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = r8p_cached_local_from_global(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                base_patch = torch.sigmoid(
                    output['base_logits'].float()
                )[0, 0].cpu().numpy()
                final_patch = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                if ablation == 'satellite':
                    final_patch = base_patch
                satellite_patch = output[
                    'satellite_logits'
                ].float()[0, 0].cpu().numpy()
                gate_value = float(
                    output['satellite_gate'].float().reshape(-1)[0].cpu()
                )
                destination = tuple(
                    slice(native_start[axis], min(
                        native_start[axis] + LOCAL_PATCH_SIZE[axis],
                        native_shape[axis],
                    ))
                    for axis in range(3)
                )
                copied_shape = tuple(
                    item.stop - item.start for item in destination
                )
                source = tuple(
                    slice(0, copied_shape[axis]) for axis in range(3)
                )
                weight = PILOT_WINDOW_WEIGHT[source]
                final_sum[destination] += final_patch[source] * weight
                base_sum[destination] += base_patch[source] * weight
                satellite_logit_sum[destination] += (
                    satellite_patch[source] * weight
                )
                gate_sum[destination] += gate_value * weight
                weight_sum[destination] += weight
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete R8 inference coverage."
        )
    probability = final_sum / weight_sum
    base_probability = base_sum / weight_sum
    satellite_logit = satellite_logit_sum / weight_sum
    satellite_gate_map = gate_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(
            case['pad_start'][axis],
            case['pad_start'][axis] + native_shape[axis],
        )
        for axis in range(3)
    )
    proposal = proposal_canvas[native_slices]

    target = case['target']
    hard_background = satellite_logit[~target]
    hard_count = min(R8P_NEGATIVE_BACKGROUND_TOPK, hard_background.size)
    if hard_count > 0:
        hard_negative_mean = float(np.partition(
            hard_background, hard_background.size - hard_count
        )[-hard_count:].mean())
    else:
        hard_negative_mean = float('nan')
    positive_mean = (
        float(satellite_logit[target].mean())
        if bool(target.any()) else float('nan')
    )
    component_records = r8t_component_satellite_records(
        satellite_logit, target, case['spacing']
    )
    if ablation == 'none':
        R8T_VALIDATION_CAPTURE.append({
            'subject_id': case['subject_id'],
            'satellite_positive_logit_mean': positive_mean,
            'satellite_hard_negative_logit_mean': hard_negative_mean,
            'satellite_positive_minus_hard_negative_margin': (
                positive_mean - hard_negative_mean
                if np.isfinite(positive_mean) and np.isfinite(hard_negative_mean)
                else float('nan')
            ),
            'satellite_gate_mean': float(satellite_gate_map.mean()),
            'satellite_gate_abs_mean': float(
                np.abs(satellite_gate_map).mean()
            ),
            'satellite_gate_minimum': float(satellite_gate_map.min()),
            'satellite_gate_maximum': float(satellite_gate_map.max()),
            'base_to_final_mean_absolute_delta': float(
                np.abs(probability - base_probability).mean()
            ),
            'base_to_final_target_mean_absolute_delta': (
                float(np.abs(
                    probability[target] - base_probability[target]
                ).mean()) if bool(target.any()) else float('nan')
            ),
            'component_records': component_records,
        })
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': target, 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def r8t_satellite_validation_summary(epoch):
    rows = []
    component_rows = []
    for item in R8T_VALIDATION_CAPTURE:
        flat = {
            key: value for key, value in item.items()
            if key != 'component_records'
        }
        flat['epoch'] = int(epoch)
        rows.append(flat)
        for component in item['component_records']:
            component_rows.append({
                'epoch': int(epoch),
                'subject_id': item['subject_id'],
                **component,
            })
    frame = pd.DataFrame(rows)
    component_frame = pd.DataFrame(component_rows)
    epoch_root = R8T_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    atomic_write_csv(frame, epoch_root / 'satellite_case_metrics.csv')
    atomic_write_csv(
        component_frame, epoch_root / 'satellite_component_metrics.csv'
    )
    finite_margins = (
        frame['satellite_positive_minus_hard_negative_margin']
        .replace([np.inf, -np.inf], np.nan).dropna()
        if len(frame) else pd.Series(dtype=float)
    )
    small_components = (
        component_frame[
            component_frame['component_tier'].astype(str).isin(
                R8T_SMALL_TIERS
            )
        ] if len(component_frame) else component_frame
    )
    return {
        'satellite_positive_minus_hard_negative_logit_margin': (
            float(finite_margins.mean()) if len(finite_margins) else 0.0
        ),
        'satellite_gate_mean': (
            float(frame['satellite_gate_mean'].mean()) if len(frame) else 0.0
        ),
        'satellite_gate_abs_mean': (
            float(frame['satellite_gate_abs_mean'].mean())
            if len(frame) else 0.0
        ),
        'satellite_gate_minimum': (
            float(frame['satellite_gate_minimum'].min()) if len(frame) else 0.0
        ),
        'satellite_gate_maximum': (
            float(frame['satellite_gate_maximum'].max()) if len(frame) else 0.0
        ),
        'base_to_final_mean_absolute_delta': (
            float(frame['base_to_final_mean_absolute_delta'].mean())
            if len(frame) else 0.0
        ),
        'base_to_final_target_mean_absolute_delta': float(
            frame['base_to_final_target_mean_absolute_delta']
            .replace([np.inf, -np.inf], np.nan).dropna().mean()
        ) if len(frame) else 0.0,
        'small_satellite_component_representation_failure_fraction': (
            float(np.mean(
                small_components['satellite_probability_maximum'] < 0.10
            )) if len(small_components) else 0.0
        ),
        'small_satellite_component_probability_maximum_mean': (
            float(small_components[
                'satellite_probability_maximum'
            ].mean()) if len(small_components) else 0.0
        ),
        'satellite_components_audited': int(len(component_frame)),
        'satellite_small_components_audited': int(len(small_components)),
    }


@torch.no_grad()
def r8t_validate(model, epoch, device):
    global PILOT_VALIDATION_ROOT, LOCAL_THRESHOLDS, PROPOSAL_THRESHOLDS
    global pilot_infer_case, R8T_VALIDATION_CAPTURE
    original_root = PILOT_VALIDATION_ROOT
    original_local_thresholds = LOCAL_THRESHOLDS
    original_proposal_thresholds = PROPOSAL_THRESHOLDS
    original_infer_case = pilot_infer_case
    R8T_VALIDATION_CAPTURE = []
    try:
        PILOT_VALIDATION_ROOT = R8T_VALIDATION_ROOT
        LOCAL_THRESHOLDS = R8P_LOCAL_THRESHOLDS
        PROPOSAL_THRESHOLDS = R8P_PROPOSAL_THRESHOLDS
        pilot_infer_case = r8t_infer_case
        summary, tier_rows = pilot_validate(model, epoch, device)
    finally:
        PILOT_VALIDATION_ROOT = original_root
        LOCAL_THRESHOLDS = original_local_thresholds
        PROPOSAL_THRESHOLDS = original_proposal_thresholds
        pilot_infer_case = original_infer_case
    satellite_summary = r8t_satellite_validation_summary(epoch)
    summary.update(satellite_summary)
    summary['small_component_recall'] = r8t_small_component_recall(tier_rows)
    epoch_root = R8T_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    return summary, tier_rows


@torch.no_grad()
def r8t_branch_probe(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['satellite', 'global', 'arterial', 'clinical']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8),
            GLOBAL_CANVAS_SHAPE, (0, 1, 2),
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_arterial = ablation == 'arterial'
            force_clinical = ablation == 'clinical'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model.base, batch,
                    force_arterial_dropout=force_arterial,
                    force_clinical_dropout=force_clinical,
                )
                output = r8p_cached_local_from_global(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_clinical_dropout=force_clinical,
                )
            logits = (
                output['base_logits']
                if ablation == 'satellite' else output['logits']
            )
            return (
                torch.sigmoid(logits.float())[0, 0].cpu().numpy(),
                torch.sigmoid(
                    cache['proposal_logits'].float()
                )[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            rows.append({
                'epoch': int(epoch),
                'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.abs(baseline_local - ablated_local).mean()
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.abs(baseline_local - ablated_local).max()
                ),
                'mean_absolute_target_probability_delta': (
                    float(np.abs(
                        baseline_local[local_target]
                        - ablated_local[local_target]
                    ).mean()) if bool(local_target.any()) else float('nan')
                ),
                'mean_absolute_proposal_probability_delta': float(
                    np.abs(baseline_proposal - ablated_proposal).mean()
                ),
            })
    frame = pd.DataFrame(rows)
    R8T_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(
        frame,
        R8T_DIAGNOSTIC_ROOT
        / f'epoch_{int(epoch):03d}_branch_reliance.csv',
    )
    if R8T_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(R8T_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, R8T_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()


def r8t_base_drift(model, r5_reference_state, epoch):
    squared_difference = 0.0
    squared_reference = 0.0
    maximum_absolute_difference = 0.0
    compared = 0
    for name, parameter in model.base.named_parameters():
        if name not in r5_reference_state:
            raise RuntimeError(f'R5 reference lacks base parameter {name}.')
        observed = parameter.detach().float().cpu()
        reference = r5_reference_state[name].detach().float().cpu()
        difference = observed - reference
        squared_difference += float(torch.sum(difference * difference))
        squared_reference += float(torch.sum(reference * reference))
        maximum_absolute_difference = max(
            maximum_absolute_difference,
            float(difference.abs().max().item()),
        )
        compared += int(parameter.numel())
    row = {
        'epoch': int(epoch),
        'parameters_compared': int(compared),
        'relative_l2_drift': float(math.sqrt(
            squared_difference / max(squared_reference, 1e-24)
        )),
        'maximum_absolute_parameter_difference': float(
            maximum_absolute_difference
        ),
    }
    frame = pd.DataFrame([row])
    if R8T_BASE_DRIFT_HISTORY_PATH.exists():
        prior = pd.read_csv(R8T_BASE_DRIFT_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, R8T_BASE_DRIFT_HISTORY_PATH)
    return row


def r8t_recommendation(best_summary, baseline, eligible):
    if not eligible:
        return (
            'Retain r5. No R8 epoch satisfied all deployment-metric guards; '
            'inspect the per-tier and satellite-calibration audits.'
        )
    if float(best_summary[
        'satellite_positive_minus_hard_negative_logit_margin'
    ]) <= 0.0:
        return (
            'Keep the R5 base and revise the satellite absolute-calibration '
            'objective before any fold replication.'
        )
    if float(best_summary['small_component_recall']) <= float(
        baseline['small_component_recall']
    ):
        return (
            'Satellite features separate positives but are not converting to '
            'detections; revise the bounded fusion/threshold contract.'
        )
    if float(best_summary['positive_mean_dice']) < float(
        baseline['r5_mean_dice']
    ):
        return (
            'Small-component recall improved at a mean-Dice cost; reduce the '
            'gate or joint-base learning rate before replication.'
        )
    return (
        'R8 is directionally successful on fold 0. Build a separate '
        'fold-replication preflight; do not open the sealed test set yet.'
    )


def run_r8_fold0_pilot_training():
    if r8t_existing_result_complete():
        completed = load_json(R8P_RESULT_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R8 FOLD-0 PILOT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Best eligible epoch:                {completed.get("best_epoch")}')
        print(f'Directional success:                '
              f'{completed.get("directional_success")}')
        print(f'Result gate:                        {R8P_RESULT_GATE_PATH}')
        print('No model checkpoint or training artifact was rewritten.')
        return

    preflight_gate = load_json(R8P_GATE_PATH)
    r8_policy = load_json(R8P_POLICY_PATH)
    resume_contract = load_json(R8P_RESUME_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_baseline = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
    r5_summary = dict(r5_baseline['validation_summary'])
    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    preflight_checks = {
        'r8_pilot_preflight_passed': (
            preflight_gate.get('preflight_passed') is True
        ),
        'r8_pilot_training_authorized': (
            preflight_gate.get('r8_fold0_pilot_training_authorized') is True
        ),
        'full_fivefold_training_locked': (
            preflight_gate.get('full_fivefold_training_authorized') is False
        ),
        'pilot_version_matches': (
            preflight_gate.get('pilot_version') == R8P_PILOT_VERSION
        ),
        'pilot_code_revision_matches': (
            preflight_gate.get('pilot_code_revision')
            == R8P_PILOT_CODE_REVISION
        ),
        'r8_architecture_gate_hash_matches': (
            preflight_gate.get('r8_architecture_gate_sha256')
            == file_sha256(R8_GATE_PATH)
        ),
        'pilot_policy_hash_matches': (
            preflight_gate.get('pilot_policy_sha256')
            == file_sha256(R8P_POLICY_PATH)
        ),
        'resume_contract_hash_is_current': (
            resume_contract.get('pilot_policy_sha256')
            == file_sha256(R8P_POLICY_PATH)
        ),
        'r5_checkpoint_hash_matches': (
            preflight_gate.get('r5_checkpoint_sha256')
            == r5_checkpoint_sha256
            == r5_result.get('best_checkpoint_sha256')
        ),
        'r7_weight_migration_disabled': (
            resume_contract.get('r7_checkpoint_migration_allowed') is False
        ),
        'fold0_partition_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_and_test_counts_match': (
            len(locked_validation_df) == 23 and len(sealed_test_df) == 23
        ),
    }
    if not all(preflight_checks.values()):
        failed = [name for name, passed in preflight_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R8 TRAINING UPSTREAM GATE FAILED. Do not train. Failed: '
            + ', '.join(failed)
        )
    if R8P_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            'An incompatible R8 result gate already exists. Refusing to '
            'overwrite it; use a new pilot version/directory.'
        )
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 R8 fold-0 training requires CUDA.')

    R8P_PILOT_ROOT.mkdir(parents=True, exist_ok=True)
    R8T_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
    R8T_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
    device = torch.device('cuda')
    random.seed(R8P_SEED)
    set_determinism(R8P_SEED)
    gc.collect()
    torch.cuda.empty_cache()

    r5_checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    r5_checkpoint_checks = {
        'pilot_version': (
            r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'code_revision': (
            r5_checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
        ),
        'parameter_count': int(
            r5_checkpoint.get('parameter_count', -1)
        ) == EXPECTED_V22_PARAMETERS,
        'ema_state': 'ema_model_state' in r5_checkpoint,
    }
    if not all(r5_checkpoint_checks.values()):
        failed = [name for name, passed in r5_checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 warm start: ' + ', '.join(failed))
    r5_reference_state = {
        name: value.detach().cpu().clone()
        for name, value in r5_checkpoint['ema_model_state'].items()
    }

    model = VascuFateNetR8(pilot_schema).to(device)
    model.load_r5_base_state(r5_reference_state)
    if sum(parameter.numel() for parameter in model.parameters()) != (
        R8P_EXPECTED_PARAMETERS
    ):
        raise RuntimeError('R8 parameter count changed before training.')
    model.reset_satellite_gate_to_zero()
    
    
    rng_before_ema_construction = pilot_capture_rng_state()
    ema_model = VascuFateNetR8(pilot_schema).to(device).eval()
    ema_model.load_state_dict(model.state_dict(), strict=True)
    pilot_restore_rng_state(rng_before_ema_construction)
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    optimizer = torch.optim.AdamW(
        r8p_optimizer_groups(model), betas=(0.9, 0.99), eps=1e-8
    )
    dataset = Trial8R8Fold0Dataset()
    expected_updates_per_epoch = int(math.ceil(
        len(dataset) / R8P_ACCUMULATION_STEPS
    ))
    if len(dataset) != 168 or expected_updates_per_epoch != 42:
        raise RuntimeError('R8 epoch/update cardinality changed.')

    baseline = {
        'r5_mean_dice': float(r5_summary['positive_mean_dice']),
        'r5_global_dice': float(r5_summary['positive_global_dice']),
        'mean_dice_floor': float(
            r8_policy['checkpoint_eligibility']['mean_dice_floor']
        ),
        'global_dice_floor': float(
            r8_policy['checkpoint_eligibility']['global_dice_floor']
        ),
        'small_case_mean_dice': r8p_r5_small_case_dice(r5_summary),
        'small_component_recall': r8p_r5_small_component_recall(r5_baseline),
    }
    pilot_config = {
        'pilot_version': R8P_PILOT_VERSION,
        'code_revision': R8P_PILOT_CODE_REVISION,
        'implementation_revision': R8T_IMPLEMENTATION_REVISION,
        'architecture': R8_MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82,
        'heldout_subjects': 20,
        'parameter_count': R8P_EXPECTED_PARAMETERS,
        'samples_per_epoch': len(dataset),
        'optimizer_updates_per_epoch': expected_updates_per_epoch,
        'maximum_epochs': R8P_MAX_EPOCHS,
        'minimum_epochs': R8P_MINIMUM_EPOCHS,
        'early_stopping_patience': R8P_EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': (
            R8P_EARLY_STOPPING_MINIMUM_DELTA
        ),
        'early_stopping_metric': 'positive_mean_dice_checked_every_epoch',
        'accumulation_steps': R8P_ACCUMULATION_STEPS,
        'gradient_clip_norm': R8P_GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'learning_rate_policy': {
            str(epoch): r8p_learning_rates(epoch)
            for epoch in range(1, R8P_MAX_EPOCHS + 1)
        },
        'stage_policy': {
            str(epoch): r8p_stage(epoch)
            for epoch in range(1, R8P_MAX_EPOCHS + 1)
        },
        'checkpoint_baseline': baseline,
        'checkpoint_selection': (
            'eligibility_then_lexicographic_mean_small_case_'
            'small_component_lesion_f1_global'
        ),
        'warm_start': 'r5_best_ema_only',
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'r7_weights_used': False,
        'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'resume_contract_sha256': file_sha256(R8P_RESUME_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(pilot_config)
    if R8P_PILOT_CONFIG_PATH.exists():
        existing = load_json(R8P_PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing R8 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(pilot_config, R8P_PILOT_CONFIG_PATH)
    if pilot_config_hash(load_json(R8P_PILOT_CONFIG_PATH)) != config_hash:
        raise RuntimeError('R8 pilot configuration round-trip is unstable.')

    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': R8P_PILOT_VERSION,
        'code_revision': R8P_PILOT_CODE_REVISION,
        'implementation_revision': R8T_IMPLEMENTATION_REVISION,
        'pilot_config_sha256': file_sha256(R8P_PILOT_CONFIG_PATH),
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_best_ema_used_for_initialization': True,
        'r7_weights_used': False,
        'same_r8_last_checkpoint_only_resume': True,
        'fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'upstream_checks': preflight_checks,
    }
    if R8P_PILOT_START_GATE_PATH.exists():
        existing_start = load_json(R8P_PILOT_START_GATE_PATH)
        immutable = [
            'pilot_version', 'code_revision', 'implementation_revision',
            'pilot_config_sha256', 'pilot_policy_sha256',
            'preflight_gate_sha256', 'r5_checkpoint_sha256',
        ]
        if any(existing_start.get(name) != start_gate.get(name) for name in immutable):
            raise RuntimeError(
                'Existing R8 start gate differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(start_gate, R8P_PILOT_START_GATE_PATH)

    start_epoch = 1
    optimizer_steps = 0
    best_selection_key = None
    best_epoch = 0
    best_thresholds = {'local': 0.50, 'proposal': 0.10}
    best_metrics = None
    patience_reference_mean_dice = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    resumed = False
    if R8P_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            R8P_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': (
                checkpoint.get('pilot_version') == R8P_PILOT_VERSION
            ),
            'code_revision': (
                checkpoint.get('code_revision') == R8P_PILOT_CODE_REVISION
            ),
            'implementation_revision': (
                checkpoint.get('implementation_revision')
                == R8T_IMPLEMENTATION_REVISION
            ),
            'policy_hash': (
                checkpoint.get('pilot_policy_sha256')
                == file_sha256(R8P_POLICY_PATH)
            ),
            'preflight_hash': (
                checkpoint.get('preflight_gate_sha256')
                == file_sha256(R8P_GATE_PATH)
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'r5_hash': (
                checkpoint.get('r5_checkpoint_sha256')
                == r5_checkpoint_sha256
            ),
            'parameters': int(
                checkpoint.get('parameter_count', -1)
            ) == R8P_EXPECTED_PARAMETERS,
            'r7_weights_excluded': checkpoint.get('r7_weights_used') is False,
            'required_fields': all(
                name in checkpoint
                for name in resume_contract['required_checkpoint_fields']
            ),
        }
        if not all(resume_checks.values()):
            failed = [name for name, passed in resume_checks.items() if not passed]
            raise RuntimeError('Unsafe R8 checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'], strict=True)
        ema_model.load_state_dict(checkpoint['ema_model_state'], strict=True)
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        optimizer_steps = int(checkpoint['optimizer_steps'])
        best_selection_key = checkpoint['best_selection_key']
        best_epoch = int(checkpoint.get('best_epoch', 0))
        best_thresholds = dict(checkpoint['best_thresholds'])
        best_metrics = checkpoint.get('best_metrics')
        patience_reference_mean_dice = float(
            checkpoint['patience_reference_mean_dice']
        )
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if not R8P_HISTORY_PATH.exists():
            raise RuntimeError('R8 resume checkpoint exists without history.')
        history_rows = pd.read_csv(R8P_HISTORY_PATH).to_dict('records')
        if max(int(row['epoch']) for row in history_rows) != start_epoch - 1:
            raise RuntimeError('R8 resume history/checkpoint epochs disagree.')
        if optimizer_steps != (start_epoch - 1) * expected_updates_per_epoch:
            raise RuntimeError('R8 resume optimizer-step count is inconsistent.')
        if best_epoch > 0 and not R8P_BEST_CHECKPOINT_PATH.exists():
            raise RuntimeError('R8 resume metadata names a missing best checkpoint.')
        resumed = True
    else:
        if R8P_BEST_CHECKPOINT_PATH.exists() or R8P_HISTORY_PATH.exists():
            raise RuntimeError(
                'R8 partial artifacts exist without a last checkpoint. '
                'Refusing an unsafe implicit restart.'
            )

    del r5_checkpoint
    gc.collect()
    torch.cuda.empty_cache()

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {R8P_PILOT_VERSION}')
    print(f'Implementation revision:             {R8T_IMPLEMENTATION_REVISION}')
    print('Warm start:                          R5 best EMA only')
    print('R7 weights used:                     False')
    print(f'Resume epoch:                         {start_epoch}')
    print(f'Same-run resume:                      {resumed}')
    print(f'Fit / held-out subjects:              82 / 20')
    print(f'Samples / optimizer updates per epoch:{len(dataset)} / '
          f'{expected_updates_per_epoch}')
    print(f'Parameters:                           {R8P_EXPECTED_PARAMETERS:,}')
    print('Validation and early-stop check:      every epoch')
    print(f'Minimum epochs / patience:            '
          f'{R8P_MINIMUM_EPOCHS} / {R8P_EARLY_STOPPING_PATIENCE}')
    print('Checkpoint policy:                    deployment-metric Pareto guard')
    print('Fold-0 held-out targets:              validation only')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Full five-fold training authorized:   False')
    print('R5 mean/global Dice:                  '
          f'{baseline["r5_mean_dice"]:.4f} / '
          f'{baseline["r5_global_dice"]:.4f}')
    print('Eligibility mean/global floors:       '
          f'{baseline["mean_dice_floor"]:.4f} / '
          f'{baseline["global_dice_floor"]:.4f}')

    resume_already_exhausted_patience = bool(
        resumed
        and start_epoch > R8P_MINIMUM_EPOCHS
        and validations_without_improvement
        >= R8P_EARLY_STOPPING_PATIENCE
    )
    stop_reason = (
        'early_stopping'
        if resume_already_exhausted_patience else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    training_epochs = (
        [] if resume_already_exhausted_patience
        else range(start_epoch, R8P_MAX_EPOCHS + 1)
    )
    for epoch in training_epochs:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True,
        )
        stage, learning_rates = r8p_apply_stage(model, epoch, optimizer)
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        satellite_training_rows = []
        dropout_counts = defaultdict(int)
        processed = 0
        epoch_gradient_norms = []
        epoch_optimizer_steps = 0
        for batch_index, cpu_batch in enumerate(loader):
            flags = r8t_dropout_flags(epoch, batch_index)
            for name, value in flags.items():
                dropout_counts[name] += int(bool(value))
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts, satellite_rows, output = (
                    r8p_compute_training_loss(
                        model, cpu_batch, device, epoch,
                        dropout_flags_override=flags,
                    )
                )
                scaled_loss = loss / R8P_ACCUMULATION_STEPS
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError(f'Epoch {epoch}: non-finite training loss.')
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            satellite_training_rows.extend(satellite_rows)
            del output, loss, scaled_loss, parts
            should_step = (
                (batch_index + 1) % R8P_ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    [parameter for parameter in model.parameters()
                     if parameter.requires_grad],
                    R8P_GRADIENT_CLIP_NORM,
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_steps += 1
                epoch_optimizer_steps += 1
                r8t_ema_update(ema_model, model, optimizer_steps)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))
        if epoch_optimizer_steps != expected_updates_per_epoch:
            raise RuntimeError(
                f'Epoch {epoch}: expected {expected_updates_per_epoch} '
                f'optimizer steps, observed {epoch_optimizer_steps}.'
            )

        summary, tier_rows = r8t_validate(ema_model, epoch, device)
        metrics = {
            'positive_mean_dice': float(summary['positive_mean_dice']),
            'positive_global_dice': float(summary['positive_global_dice']),
            'small_case_mean_dice': float(summary['small_case_mean_dice']),
            'small_component_recall': float(
                summary['small_component_recall']
            ),
            'lesion_f1': float(summary['lesion_f1']),
        }
        eligible, eligibility_guards, selection_key = (
            r8p_checkpoint_evaluation(metrics, baseline)
        )
        strict_best_improved = bool(
            eligible
            and (
                best_selection_key is None
                or tuple(selection_key) > tuple(best_selection_key)
            )
        )
        if strict_best_improved:
            best_selection_key = list(selection_key)
            best_epoch = int(epoch)
            best_thresholds = {
                'local': float(summary['threshold']),
                'proposal': float(summary['proposal_threshold']),
            }
            best_metrics = dict(metrics)

        current_mean_dice = float(summary['positive_mean_dice'])
        patience_improved = (
            current_mean_dice
            > patience_reference_mean_dice
            + R8P_EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_improved:
            patience_reference_mean_dice = current_mean_dice
            validations_without_improvement = 0
        elif epoch < R8P_MINIMUM_EPOCHS:
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1

        branch_frame = pd.DataFrame()
        drift = {
            'relative_l2_drift': 0.0,
            'maximum_absolute_parameter_difference': 0.0,
        }
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = r8t_branch_probe(ema_model, epoch, device)
            drift = r8t_base_drift(
                ema_model, r5_reference_state, epoch
            )

        elapsed_minutes = (time.time() - epoch_start) / 60.0
        satellite_training = {
            'train_satellite_positive_logit_mean': float(np.mean([
                row['positive_logit_mean'] for row in satellite_training_rows
            ])) if satellite_training_rows else float('nan'),
            'train_satellite_hard_negative_logit_mean': float(np.mean([
                row['hard_negative_logit_mean']
                for row in satellite_training_rows
            ])) if satellite_training_rows else float('nan'),
            'train_satellite_relative_logit_margin': float(np.mean([
                row['relative_logit_margin']
                for row in satellite_training_rows
            ])) if satellite_training_rows else float('nan'),
        }
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        history_row = {
            'epoch': int(epoch),
            'stage': stage,
            **{
                f'train_{name}': value / max(processed, 1)
                for name, value in running.items()
            },
            **satellite_training,
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'optimizer_steps_this_epoch': int(epoch_optimizer_steps),
            'optimizer_steps_total': int(optimizer_steps),
            **{
                f'learning_rate_{name}': float(value)
                for name, value in learning_rates.items()
            },
            **{
                f'dropout_count_{name}': int(value)
                for name, value in dropout_counts.items()
            },
            'validation_mean_dice': metrics['positive_mean_dice'],
            'validation_median_dice': float(
                summary['positive_median_dice']
            ),
            'validation_global_dice': metrics['positive_global_dice'],
            'validation_small_case_mean_dice': metrics[
                'small_case_mean_dice'
            ],
            'validation_lesion_precision': float(
                summary['lesion_precision']
            ),
            'validation_lesion_recall': float(summary['lesion_recall']),
            'validation_lesion_f1': metrics['lesion_f1'],
            'validation_component_recall': float(
                summary['component_recall']
            ),
            'validation_small_component_recall': metrics[
                'small_component_recall'
            ],
            'validation_satellite_margin': float(summary[
                'satellite_positive_minus_hard_negative_logit_margin'
            ]),
            'validation_satellite_gate_abs_mean': float(
                summary['satellite_gate_abs_mean']
            ),
            'validation_small_satellite_representation_failure': float(
                summary[
                    'small_satellite_component_representation_failure_fraction'
                ]
            ),
            'validation_base_to_final_target_delta': float(
                summary['base_to_final_target_mean_absolute_delta']
            ),
            **{
                f'validation_{name}_mean_dice': float(
                    bin_metrics.get(name, float('nan'))
                )
                for name in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'local_threshold': float(summary['threshold']),
            'proposal_threshold': float(summary['proposal_threshold']),
            'checkpoint_eligible': bool(eligible),
            **{
                f'eligibility_{name}': bool(value)
                for name, value in eligibility_guards.items()
            },
            'strict_eligible_best_improved': bool(strict_best_improved),
            'best_epoch': int(best_epoch),
            'patience_improved_by_minimum_delta': bool(patience_improved),
            'patience_reference_mean_dice': float(
                patience_reference_mean_dice
            ),
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'r5_base_relative_l2_drift': float(
                drift['relative_l2_drift']
            ),
            'elapsed_minutes': float(elapsed_minutes),
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != int(epoch)
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), R8P_HISTORY_PATH)
        satellite_history = pd.DataFrame([{
            'epoch': int(epoch),
            **satellite_training,
            'validation_satellite_margin': float(summary[
                'satellite_positive_minus_hard_negative_logit_margin'
            ]),
            'validation_gate_mean': float(summary['satellite_gate_mean']),
            'validation_gate_abs_mean': float(
                summary['satellite_gate_abs_mean']
            ),
            'validation_gate_minimum': float(
                summary['satellite_gate_minimum']
            ),
            'validation_gate_maximum': float(
                summary['satellite_gate_maximum']
            ),
            'validation_small_representation_failure': float(
                summary[
                    'small_satellite_component_representation_failure_fraction'
                ]
            ),
            'validation_base_to_final_target_delta': float(
                summary['base_to_final_target_mean_absolute_delta']
            ),
        }])
        if R8T_SATELLITE_HISTORY_PATH.exists():
            prior = pd.read_csv(R8T_SATELLITE_HISTORY_PATH)
            prior = prior[prior['epoch'].astype(int) != int(epoch)]
            satellite_history = pd.concat(
                [prior, satellite_history], ignore_index=True
            )
        atomic_write_csv(satellite_history, R8T_SATELLITE_HISTORY_PATH)

        checkpoint_record = {
            'pilot_version': R8P_PILOT_VERSION,
            'code_revision': R8P_PILOT_CODE_REVISION,
            'implementation_revision': R8T_IMPLEMENTATION_REVISION,
            'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
            'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
            'config_hash': config_hash,
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'r7_weights_used': False,
            'parameter_count': R8P_EXPECTED_PARAMETERS,
            'epoch': int(epoch),
            'stage': stage,
            'optimizer_steps': int(optimizer_steps),
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_selection_key': best_selection_key,
            'best_epoch': int(best_epoch),
            'best_thresholds': best_thresholds,
            'best_metrics': best_metrics,
            'patience_reference_mean_dice': float(
                patience_reference_mean_dice
            ),
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'validation_summary': summary,
            'checkpoint_eligible': bool(eligible),
            'eligibility_guards': eligibility_guards,
            'selection_key': list(selection_key),
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }
        pilot_atomic_torch_save(
            checkpoint_record, R8P_LAST_CHECKPOINT_PATH
        )
        if strict_best_improved:
            pilot_atomic_torch_save(
                checkpoint_record, R8P_BEST_CHECKPOINT_PATH
            )
        atomic_write_json({
            'pilot_version': R8P_PILOT_VERSION,
            'code_revision': R8P_PILOT_CODE_REVISION,
            'implementation_revision': R8T_IMPLEMENTATION_REVISION,
            'status': 'running',
            'last_completed_epoch': int(epoch),
            'stage': stage,
            'optimizer_steps': int(optimizer_steps),
            'best_epoch': int(best_epoch),
            'eligible_best_found': best_epoch > 0,
            'best_selection_key': best_selection_key,
            'best_thresholds': best_thresholds,
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'validation_checked_this_epoch': True,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, R8T_STATE_PATH)

        train_margin = satellite_training[
            'train_satellite_relative_logit_margin'
        ]
        print(
            f'Epoch {epoch:03d}/{R8P_MAX_EPOCHS}: stage={stage}, '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'satellite={running["satellite_objective"]/max(processed,1):.4f}, '
            f'train_margin={train_margin:.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean/median/global Dice='
            f'{summary["positive_mean_dice"]:.4f}/'
            f'{summary["positive_median_dice"]:.4f}/'
            f'{summary["positive_global_dice"]:.4f}, '
            f'lesion P/R/F1={summary["lesion_precision"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}/'
            f'{summary["lesion_f1"]:.4f}'
        )
        print(
            f'  Small cases/components: Dice='
            f'{summary["small_case_mean_dice"]:.4f}, recall='
            f'{summary["small_component_recall"]:.4f}; '
            f'all-component recall={summary["component_recall"]:.4f}'
        )
        print(
            '  Satellite: held-out margin/gate|mean|/small repr. failure='
            f'{summary["satellite_positive_minus_hard_negative_logit_margin"]:.4f}/'
            f'{summary["satellite_gate_abs_mean"]:.5f}/'
            f'{summary["small_satellite_component_representation_failure_fraction"]:.4f}, '
            f'base-to-final target delta='
            f'{summary["base_to_final_target_mean_absolute_delta"]:.5f}'
        )
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}; '
            f'eligible={eligible}, guards={eligibility_guards}, '
            f'best_epoch={best_epoch}, patience='
            f'{validations_without_improvement}/'
            f'{R8P_EARLY_STOPPING_PATIENCE}'
        )
        if len(branch_frame):
            branch_delta = branch_frame.groupby('ablation')[
                'mean_absolute_target_probability_delta'
            ].mean().to_dict()
            print(
                '  Branch target deltas: '
                + ', '.join(
                    f'{name}={float(value):.5f}'
                    for name, value in sorted(branch_delta.items())
                )
                + f'; R5-base relative drift='
                f'{drift["relative_l2_drift"]:.6f}'
            )
        last_summary, last_tier_rows = summary, tier_rows
        if (
            epoch >= R8P_MINIMUM_EPOCHS
            and validations_without_improvement
            >= R8P_EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if last_summary is None:
        if not R8P_LAST_CHECKPOINT_PATH.exists():
            raise RuntimeError('R8 training produced no checkpoint.')
        last_checkpoint = torch.load(
            R8P_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        last_summary = dict(last_checkpoint['validation_summary'])
        last_epoch = int(last_checkpoint['epoch'])
        last_tier_path = (
            R8T_VALIDATION_ROOT / f'epoch_{last_epoch:03d}'
            / 'component_tier_metrics.csv'
        )
        if not last_tier_path.exists():
            raise RuntimeError(
                'R8 last checkpoint lacks its component-tier audit.'
            )
        last_tier_rows = pd.read_csv(last_tier_path).to_dict('records')

    eligible_best_found = bool(
        best_epoch > 0 and R8P_BEST_CHECKPOINT_PATH.exists()
    )
    if eligible_best_found:
        best_checkpoint = torch.load(
            R8P_BEST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        ema_model.load_state_dict(
            best_checkpoint['ema_model_state'], strict=True
        )
        best_summary = dict(best_checkpoint['validation_summary'])
        best_epoch = int(best_checkpoint['epoch'])
        best_tier_path = (
            R8T_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
            / 'component_tier_metrics.csv'
        )
        best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
        best_branch_path = (
            R8T_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        )
        if best_branch_path.exists():
            best_branch_frame = pd.read_csv(best_branch_path)
        else:
            best_branch_frame = r8t_branch_probe(
                ema_model, best_epoch, device
            )
        best_drift = r8t_base_drift(
            ema_model, r5_reference_state, best_epoch
        )
    else:
        best_summary = dict(last_summary)
        best_tier_rows = list(last_tier_rows)
        best_branch_frame = pd.DataFrame()
        best_drift = {
            'relative_l2_drift': None,
            'maximum_absolute_parameter_difference': None,
        }

    directional_checks = {
        'eligible_r8_checkpoint_found': eligible_best_found,
        'mean_dice_not_below_r5': (
            eligible_best_found
            and float(best_summary['positive_mean_dice'])
            >= baseline['r5_mean_dice']
        ),
        'global_dice_meets_pareto_floor': (
            eligible_best_found
            and float(best_summary['positive_global_dice'])
            >= baseline['global_dice_floor']
        ),
        'small_case_dice_not_below_r5': (
            eligible_best_found
            and float(best_summary['small_case_mean_dice'])
            >= baseline['small_case_mean_dice']
        ),
        'small_component_recall_improved': (
            eligible_best_found
            and float(best_summary['small_component_recall'])
            > baseline['small_component_recall']
        ),
        'satellite_margin_is_positive': (
            eligible_best_found
            and float(best_summary[
                'satellite_positive_minus_hard_negative_logit_margin'
            ]) > 0.0
        ),
        'satellite_gate_is_active_and_bounded': (
            eligible_best_found
            and float(best_summary['satellite_gate_abs_mean']) > 1e-4
            and max(
                abs(float(best_summary['satellite_gate_minimum'])),
                abs(float(best_summary['satellite_gate_maximum'])),
            ) <= R8_SATELLITE_GATE_BOUND + 1e-5
        ),
    }
    directional_success = all(directional_checks.values())
    recommendation = r8t_recommendation(
        best_summary, baseline, eligible_best_found
    )
    branch_summary = (
        best_branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
        if len(best_branch_frame) else {}
    )
    completed_epoch = int(max(
        int(row['epoch']) for row in history_rows
    )) if history_rows else int(start_epoch - 1)
    result = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_complete': True,
        'pilot_version': R8P_PILOT_VERSION,
        'code_revision': R8P_PILOT_CODE_REVISION,
        'implementation_revision': R8T_IMPLEMENTATION_REVISION,
        'stop_reason': stop_reason,
        'last_completed_epoch': completed_epoch,
        'best_epoch': int(best_epoch) if eligible_best_found else None,
        'eligible_r8_checkpoint_found': eligible_best_found,
        'directional_success': bool(directional_success),
        'directional_checks': directional_checks,
        'best_validation_summary': best_summary,
        'best_component_tier_metrics': best_tier_rows,
        'best_branch_target_probability_deltas': branch_summary,
        'best_r5_base_parameter_drift': best_drift,
        'r5_baseline': baseline,
        'mean_dice_change_vs_r5': (
            float(best_summary['positive_mean_dice'])
            - baseline['r5_mean_dice']
        ) if eligible_best_found else None,
        'global_dice_change_vs_r5': (
            float(best_summary['positive_global_dice'])
            - baseline['r5_global_dice']
        ) if eligible_best_found else None,
        'small_case_dice_change_vs_r5': (
            float(best_summary['small_case_mean_dice'])
            - baseline['small_case_mean_dice']
        ) if eligible_best_found else None,
        'small_component_recall_change_vs_r5': (
            float(best_summary['small_component_recall'])
            - baseline['small_component_recall']
        ) if eligible_best_found else None,
        'recommendation': recommendation,
        'pilot_policy_sha256': file_sha256(R8P_POLICY_PATH),
        'preflight_gate_sha256': file_sha256(R8P_GATE_PATH),
        'pilot_config_sha256': file_sha256(R8P_PILOT_CONFIG_PATH),
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'last_checkpoint_sha256': file_sha256(R8P_LAST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': (
            file_sha256(R8P_BEST_CHECKPOINT_PATH)
            if eligible_best_found else None
        ),
        'deployment_fallback_checkpoint': (
            str(R8P_BEST_CHECKPOINT_PATH)
            if eligible_best_found else str(R5_BEST_CHECKPOINT_PATH)
        ),
        'fold0_heldout_targets_read_for_validation': True,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
        'fold_replication_preflight_allowed': bool(directional_success),
    }
    atomic_write_json(result, R8T_FINAL_SUMMARY_PATH)
    result['final_summary_sha256'] = file_sha256(R8T_FINAL_SUMMARY_PATH)
    atomic_write_json(result, R8P_RESULT_GATE_PATH)

    print('\n' + '=' * 80)
    print('TRIAL8 R8 DUAL-RESOLUTION SATELLITE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Stop reason / last epoch:            {stop_reason} / {completed_epoch}')
    print(f'Eligible R8 checkpoint found:        {eligible_best_found}')
    print(f'Best eligible epoch:                 '
          f'{best_epoch if eligible_best_found else "none"}')
    print(f'Best mean/global Dice:               '
          f'{float(best_summary["positive_mean_dice"]):.4f} / '
          f'{float(best_summary["positive_global_dice"]):.4f}')
    print(f'Best small-case Dice / comp. recall: '
          f'{float(best_summary["small_case_mean_dice"]):.4f} / '
          f'{float(best_summary["small_component_recall"]):.4f}')
    print('R5 mean/global Dice:                 '
          f'{baseline["r5_mean_dice"]:.4f} / '
          f'{baseline["r5_global_dice"]:.4f}')
    print('Satellite margin / |gate| mean:      '
          f'{float(best_summary["satellite_positive_minus_hard_negative_logit_margin"]):.4f} / '
          f'{float(best_summary["satellite_gate_abs_mean"]):.5f}')
    print(f'Directional success:                 {directional_success}')
    print(f'Recommended next action:             {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print(f'Best checkpoint:                     '
          f'{R8P_BEST_CHECKPOINT_PATH if eligible_best_found else "none; retain R5"}')
    print(f'Last checkpoint:                     {R8P_LAST_CHECKPOINT_PATH}')
    print(f'History:                             {R8P_HISTORY_PATH}')
    print(f'Result gate:                         {R8P_RESULT_GATE_PATH}')






























R9_MODEL_VERSION = 'trial8_vascufate_net_v2_4_component_rescue'
R9_PREFLIGHT_VERSION = 'v1_frozen_r5_center_extent_rescue_contract'
R9_PILOT_VERSION = 'v2_4_fold0_component_center_extent_rescue_r9'
R9_CODE_REVISION = 'r1_dense_center_exact_extent_frozen_r5_union'
R9_SEED = 20261009

R9_MAX_EPOCHS = 24
R9_MINIMUM_EPOCHS = 8
R9_EARLY_STOPPING_PATIENCE = 6
R9_EARLY_STOPPING_MINIMUM_DELTA = 0.001
R9_ACCUMULATION_STEPS = 4
R9_PEAK_LR = 1.2e-4
R9_MINIMUM_LR_FRACTION = 0.10
R9_WEIGHT_DECAY = 5.0e-4
R9_GRADIENT_CLIP_NORM = 2.0
R9_CENTER_SIGMA_MM = 2.5
R9_CENTER_RADIUS_MM = 8.0
R9_OTHER_COMPONENT_IGNORE_MM = 2.0
R9_CENTER_HARD_NEGATIVE_TOPK = 4096
R9_EXTENT_HARD_NEGATIVE_TOPK = 4096
R9_VALIDATION_OVERLAP = 0.50
R9_LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
R9_RESCUE_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)
R9_PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
R9_SMALL_TIERS = {
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
}

R9_PREFLIGHT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / R9_MODEL_VERSION / R9_PREFLIGHT_VERSION
)
R9_POLICY_PATH = R9_PREFLIGHT_ROOT / 'r9_training_policy.json'
R9_TARGET_AUDIT_PATH = R9_PREFLIGHT_ROOT / 'r9_target_geometry_audit.csv'
R9_PARAMETER_AUDIT_PATH = R9_PREFLIGHT_ROOT / 'r9_parameter_audit.csv'
R9_OUTPUT_AUDIT_PATH = R9_PREFLIGHT_ROOT / 'r9_output_shape_audit.csv'
R9_LOSS_AUDIT_PATH = R9_PREFLIGHT_ROOT / 'r9_loss_smoke_audit.csv'
R9_GRADIENT_AUDIT_PATH = R9_PREFLIGHT_ROOT / 'r9_gradient_audit.csv'
R9_CHECKS_PATH = R9_PREFLIGHT_ROOT / 'r9_preflight_checks.csv'
R9_PREFLIGHT_GATE_PATH = R9_PREFLIGHT_ROOT / 'r9_preflight_gate.json'
R9_PREFLIGHT_OUTPUTS = [
    R9_POLICY_PATH, R9_TARGET_AUDIT_PATH, R9_PARAMETER_AUDIT_PATH,
    R9_OUTPUT_AUDIT_PATH, R9_LOSS_AUDIT_PATH, R9_GRADIENT_AUDIT_PATH,
    R9_CHECKS_PATH,
]

R9_PILOT_ROOT = (
    PROJECT_ROOT / 'checkpoints' / R9_MODEL_VERSION / R9_PILOT_VERSION
)
R9_CONFIG_PATH = R9_PILOT_ROOT / 'pilot_config.json'
R9_START_GATE_PATH = R9_PILOT_ROOT / 'pilot_start_gate.json'
R9_LAST_CHECKPOINT_PATH = R9_PILOT_ROOT / 'last_checkpoint.pt'
R9_BEST_CHECKPOINT_PATH = R9_PILOT_ROOT / 'best_checkpoint.pt'
R9_DIAGNOSTIC_BEST_CHECKPOINT_PATH = (
    R9_PILOT_ROOT / 'diagnostic_best_checkpoint.pt'
)
R9_HISTORY_PATH = R9_PILOT_ROOT / 'training_history.csv'
R9_STATE_PATH = R9_PILOT_ROOT / 'pilot_state.json'
R9_RESULT_GATE_PATH = R9_PILOT_ROOT / 'pilot_result_gate.json'
R9_FINAL_SUMMARY_PATH = R9_PILOT_ROOT / 'final_pilot_summary.json'
R9_VALIDATION_ROOT = R9_PILOT_ROOT / 'validation'
R9_DIAGNOSTIC_ROOT = R9_PILOT_ROOT / 'diagnostics'

R9_NO_DROPOUT = {
    'force_atlas_dropout': False,
    'force_vascular_dropout': False,
    'force_occlusion_dropout': False,
    'force_clinical_dropout': False,
    'force_global_dropout': False,
    'force_arterial_dropout': False,
}


def r9_model_digest(module):
    digest = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        digest.update(name.encode('utf-8'))
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def r9_optimizer_lr(epoch):
    epoch = int(epoch)
    if epoch <= 2:
        return float(R9_PEAK_LR * epoch / 2.0)
    progress = np.clip(
        (epoch - 2) / max(R9_MAX_EPOCHS - 2, 1), 0.0, 1.0
    )
    minimum = R9_PEAK_LR * R9_MINIMUM_LR_FRACTION
    cosine = 0.5 * (1.0 + math.cos(math.pi * float(progress)))
    return float(minimum + (R9_PEAK_LR - minimum) * cosine)


def r9_native_domain(row, local_origin):
    subject_id = str(row['subject_id'])
    native_shape = tuple(
        int(value) for value in pilot_mask_records[subject_id]['shape']
    )
    domain_canvas, _ = symmetric_pad(
        np.ones(native_shape, dtype=np.uint8),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    return crop_at(
        domain_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)


def r9_component_targets(row, local_origin, component_id):
    subject_id = str(row['subject_id'])
    _, _, support, spacing = r7d_component_support(row)
    labels, count = ndimage.label(
        support, structure=np.ones((3, 3, 3), dtype=np.uint8)
    )
    expected = len(component_records.get(subject_id, []))
    if int(count) != int(expected):
        raise RuntimeError(
            f'{subject_id}: R9 reconstructed {count} components; '
            f'expected {expected}.'
        )
    component_id = int(component_id)
    if component_id < 0 or component_id > count:
        raise RuntimeError(f'{subject_id}: invalid R9 component {component_id}.')
    selected = (
        labels == component_id
        if component_id > 0 else np.zeros_like(support, dtype=bool)
    )
    selected_canvas, start_a = symmetric_pad(
        selected.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    support_canvas, start_b = symmetric_pad(
        support.astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    domain_canvas, start_c = symmetric_pad(
        np.ones_like(support, dtype=np.uint8),
        GLOBAL_CANVAS_SHAPE, (0, 1, 2),
    )
    if not (start_a == start_b == start_c):
        raise RuntimeError(f'{subject_id}: R9 target padding disagreement.')
    exact = crop_at(
        selected_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    full = crop_at(
        support_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    domain = crop_at(
        domain_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(bool)
    if component_id > 0 and not bool(exact.any()):
        raise RuntimeError(f'{subject_id}: R9 exact component is empty.')

    center = np.zeros(LOCAL_PATCH_SIZE, dtype=np.float32)
    if bool(exact.any()):
        distance = ndimage.distance_transform_edt(
            ~exact, sampling=tuple(float(value) for value in spacing)
        )
        center = np.exp(
            -0.5 * np.square(distance / R9_CENTER_SIGMA_MM)
        ).astype(np.float32)
        center[distance > R9_CENTER_RADIUS_MM] = 0.0
        center[exact] = 1.0

    other = full & ~exact
    if bool(other.any()):
        other_distance = ndimage.distance_transform_edt(
            ~other, sampling=tuple(float(value) for value in spacing)
        )
        other_ignore = other_distance <= R9_OTHER_COMPONENT_IGNORE_MM
    else:
        other_ignore = np.zeros_like(other, dtype=bool)
    
    
    
    other_ignore &= ~exact
    supervision_valid = domain & ~other_ignore
    if bool(exact.any()) and not bool(supervision_valid[exact].all()):
        raise RuntimeError(
            f'{subject_id}: R9 exact target was masked from supervision.'
        )
    if bool(((center > 0) & ~domain).any()):
        raise RuntimeError(f'{subject_id}: R9 center target left native domain.')
    return {
        'center_target': center[None],
        'extent_target': exact.astype(np.float32)[None],
        'supervision_valid': supervision_valid[None],
        'native_domain': domain[None],
        'other_component_ignore': other_ignore[None],
        'component_positive': np.asarray(
            component_id > 0, dtype=bool
        ),
        'component_patch_voxels': int(exact.sum()),
        'center_support_voxels': int((center > 0.05).sum()),
        'ignored_other_component_voxels': int(other_ignore.sum()),
        'center_maximum': float(center.max(initial=0.0)),
        'spacing': tuple(float(value) for value in spacing),
    }


class NativeComponentRescueHead(nn.Module):
    def __init__(self, image_channels=15, decoder_channels=32):
        super().__init__()
        self.image_stem = nn.Sequential(
            nn.Conv3d(image_channels, 8, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(8), 8), nn.GELU(),
        )
        self.context_stem = nn.Sequential(
            nn.Conv3d(decoder_channels + 2, 12, 1, bias=False),
            nn.GroupNorm(group_count(12), 12), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Conv3d(20, 24, 1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            ResidualBlock3D(24, 24, dropout=0.10),
            nn.Dropout3d(0.10),
        )
        self.center_output = nn.Conv3d(24, 1, 1)
        self.extent_output = nn.Conv3d(24, 1, 1)
        with torch.no_grad():
            self.center_output.bias.fill_(-2.2)
            self.extent_output.bias.fill_(-2.2)

    def forward(
        self, image, decoder_feature, coarse_proposal, arterial_local,
    ):
        image_feature = self.image_stem(image)
        context_feature = self.context_stem(torch.cat([
            decoder_feature,
            torch.sigmoid(coarse_proposal),
            arterial_local,
        ], dim=1))
        feature = self.fusion(torch.cat([
            image_feature, context_feature
        ], dim=1))
        center_logits = self.center_output(feature)
        extent_logits = self.extent_output(feature)
        rescue_probability = torch.sqrt(torch.clamp(
            torch.sigmoid(center_logits) * torch.sigmoid(extent_logits),
            min=0.0, max=1.0,
        ))
        return center_logits, extent_logits, rescue_probability


class VascuFateNetR9(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.base = VascuFateNetV22(schema)
        self.rescue_head = NativeComponentRescueHead()
        self._decoder_feature = None
        self._decoder_hook = self.base.local_model.decoder1.register_forward_hook(
            self._capture_decoder_feature
        )
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def _capture_decoder_feature(self, module, inputs, output):
        self._decoder_feature = output

    def load_r5_base_state(self, state):
        self.base.load_state_dict(state, strict=True)
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)
        self.base.eval()

    def train(self, mode=True):
        super().train(mode)
        self.base.eval()
        return self

    def forward(self, **kwargs):
        self._decoder_feature = None
        output = self.base(**kwargs)
        if self._decoder_feature is None:
            raise RuntimeError('R9 decoder feature hook did not fire.')
        arterial_local = sample_global_to_local(
            output['arterial_global_map'][:, None],
            kwargs['local_origin_in_canvas'],
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        if bool(kwargs.get('force_global_dropout', False)):
            coarse = torch.zeros_like(output['global_proposal_local'])
            arterial_local = torch.zeros_like(arterial_local)
        else:
            coarse = output['global_proposal_local']
        center_logits, extent_logits, rescue_probability = self.rescue_head(
            kwargs['image'], self._decoder_feature, coarse, arterial_local
        )
        base_logits = output['logits']
        self._decoder_feature = None
        return {
            **output,
            'base_logits': base_logits,
            'logits': base_logits,
            'rescue_center_logits': center_logits,
            'rescue_extent_logits': extent_logits,
            'rescue_probability': rescue_probability,
        }


class Trial8R9Fold0Dataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = r8_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = dict(self.schedule[int(index)])
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, _ = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(
            row, local_origin, entry['sample_type'],
            entry['atomic_component_id'], entry['component_tier'],
        )
        positive = str(entry['sample_type']) == 'positive'
        component_id = int(entry['atomic_component_id']) if positive else 0
        rescue = r9_component_targets(row, local_origin, component_id)
        if positive and not bool((targets['target_soft'] > 0).any()):
            raise RuntimeError(
                f'{subject_id}: R9 positive patch lost target support.'
            )
        if not positive and bool((targets['target_soft'] > 0).any()):
            raise RuntimeError(
                f'{subject_id}: R9 negative patch contains target.'
            )
        rng = np.random.default_rng(
            R9_SEED + self.epoch * 10_000_019 + int(index) * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'rescue_center_target': rescue['center_target'],
            'rescue_extent_target': rescue['extent_target'],
            'rescue_supervision_valid': rescue['supervision_valid'],
            'rescue_component_positive': rescue['component_positive'],
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
            'component_patch_voxels': np.asarray(
                rescue['component_patch_voxels'], dtype=np.int64
            ),
            'center_support_voxels': np.asarray(
                rescue['center_support_voxels'], dtype=np.int64
            ),
            'ignored_other_component_voxels': np.asarray(
                rescue['ignored_other_component_voxels'], dtype=np.int64
            ),
        })
        return sample


def r9_hard_negative_values(logits, mask, maximum_count):
    values = logits[mask]
    if values.numel() == 0:
        return logits.reshape(-1)[:0]
    count = min(int(maximum_count), int(values.numel()))
    return torch.topk(values, k=count, largest=True).values


def r9_masked_soft_dice(probability, target, mask, epsilon=1.0):
    probability = probability[mask]
    target = target[mask]
    if probability.numel() == 0:
        return probability.sum() * 0.0
    intersection = torch.sum(probability * target)
    return 1.0 - (
        2.0 * intersection + epsilon
    ) / (probability.sum() + target.sum() + epsilon)


def r9_component_rescue_loss(output, batch):
    center_logits = output['rescue_center_logits'].float()
    extent_logits = output['rescue_extent_logits'].float()
    center_target = batch['rescue_center_target'].float()
    extent_target = batch['rescue_extent_target'].float()
    valid = batch['rescue_supervision_valid'].bool()
    positive_cases = batch['rescue_component_positive'].bool().reshape(-1)
    losses = []
    rows = []
    for batch_index in range(center_logits.shape[0]):
        center_logit = center_logits[batch_index]
        extent_logit = extent_logits[batch_index]
        center_truth = center_target[batch_index]
        extent_truth = extent_target[batch_index]
        valid_mask = valid[batch_index]
        is_positive = bool(positive_cases[batch_index])
        exact = (extent_truth > 0.5) & valid_mask
        halo = (
            (center_truth >= 0.05) & ~exact & valid_mask
        )
        background = (center_truth < 0.05) & valid_mask
        center_negative = r9_hard_negative_values(
            center_logit, background, R9_CENTER_HARD_NEGATIVE_TOPK
        )
        extent_negative = r9_hard_negative_values(
            extent_logit, background, R9_EXTENT_HARD_NEGATIVE_TOPK
        )
        if center_negative.numel() == 0 or extent_negative.numel() == 0:
            raise RuntimeError('R9 loss received no valid background.')

        center_negative_loss = F.softplus(center_negative).mean()
        extent_negative_loss = F.softplus(extent_negative).mean()
        if is_positive:
            if not bool(exact.any()) or not bool(halo.any()):
                raise RuntimeError('R9 positive loss has empty exact/halo target.')
            center_exact_bce = F.binary_cross_entropy_with_logits(
                center_logit[exact], center_truth[exact]
            )
            center_halo_bce = F.binary_cross_entropy_with_logits(
                center_logit[halo], center_truth[halo]
            )
            center_selection = exact | halo
            hard_center_mask = torch.zeros_like(valid_mask)
            background_indices = torch.nonzero(
                background, as_tuple=False
            )
            if background_indices.shape[0] > 0:
                values = center_logit[background]
                count = min(
                    R9_CENTER_HARD_NEGATIVE_TOPK, int(values.numel())
                )
                top_indices = torch.topk(values, count).indices
                selected_indices = background_indices[top_indices]
                hard_center_mask[
                    selected_indices[:, 0],
                    selected_indices[:, 1],
                    selected_indices[:, 2],
                    selected_indices[:, 3],
                ] = True
            center_dice = r9_masked_soft_dice(
                torch.sigmoid(center_logit), center_truth,
                center_selection | hard_center_mask,
            )
            extent_positive_bce = F.softplus(-extent_logit[exact]).mean()
            hard_extent_mask = torch.zeros_like(valid_mask)
            background_indices = torch.nonzero(
                background, as_tuple=False
            )
            if background_indices.shape[0] > 0:
                values = extent_logit[background]
                count = min(
                    R9_EXTENT_HARD_NEGATIVE_TOPK, int(values.numel())
                )
                top_indices = torch.topk(values, count).indices
                selected_indices = background_indices[top_indices]
                hard_extent_mask[
                    selected_indices[:, 0],
                    selected_indices[:, 1],
                    selected_indices[:, 2],
                    selected_indices[:, 3],
                ] = True
            extent_dice = r9_masked_soft_dice(
                torch.sigmoid(extent_logit), extent_truth,
                exact | hard_extent_mask,
            )
            rank = F.relu(
                1.0 - center_logit[exact].mean()
                + center_negative.mean()
            )
            consistency_mask = exact | halo | hard_center_mask
            consistency = F.relu(
                torch.sigmoid(extent_logit[consistency_mask])
                - torch.sigmoid(center_logit[consistency_mask])
            ).mean()
            loss = (
                0.16 * center_exact_bce
                + 0.12 * center_halo_bce
                + 0.10 * center_negative_loss
                + 0.12 * center_dice
                + 0.16 * extent_positive_bce
                + 0.12 * extent_negative_loss
                + 0.12 * extent_dice
                + 0.07 * rank
                + 0.03 * consistency
            )
            exact_rescue = output['rescue_probability'][batch_index][exact]
            center_margin = center_logit[exact].mean() - center_negative.mean()
            extent_margin = extent_logit[exact].mean() - extent_negative.mean()
            rows.append({
                'positive_case': True,
                'exact_voxels': int(exact.sum().item()),
                'halo_voxels': int(halo.sum().item()),
                'center_margin': float(center_margin.detach().cpu()),
                'extent_margin': float(extent_margin.detach().cpu()),
                'exact_rescue_probability_mean': float(
                    exact_rescue.mean().detach().cpu()
                ),
                'center_dice_loss': float(center_dice.detach().cpu()),
                'extent_dice_loss': float(extent_dice.detach().cpu()),
                'rank_loss': float(rank.detach().cpu()),
            })
        else:
            loss = 0.50 * center_negative_loss + 0.50 * extent_negative_loss
            rows.append({
                'positive_case': False,
                'exact_voxels': 0,
                'halo_voxels': 0,
                'center_margin': float('nan'),
                'extent_margin': float('nan'),
                'exact_rescue_probability_mean': float('nan'),
                'center_dice_loss': 0.0,
                'extent_dice_loss': 0.0,
                'rank_loss': 0.0,
            })
        losses.append(loss)
        rows[-1].update({
            'center_hard_negative_logit_mean': float(
                center_negative.mean().detach().cpu()
            ),
            'extent_hard_negative_logit_mean': float(
                extent_negative.mean().detach().cpu()
            ),
            'loss': float(loss.detach().cpu()),
        })
    return torch.stack(losses).mean(), rows


def r9_training_loss(model, cpu_batch, device, dropout_flags=None):
    batch = pilot_to_device(cpu_batch, device)
    flags = R9_NO_DROPOUT if dropout_flags is None else {
        name: bool(value) for name, value in dropout_flags.items()
    }
    if set(flags) != set(R9_NO_DROPOUT):
        raise RuntimeError('R9 dropout flag schema changed.')
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22},
        **flags,
    )
    loss, rows = r9_component_rescue_loss(output, batch)
    return loss, rows, output


def r9_cached_local_from_global(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    model._decoder_feature = None
    base_output = pilot_local_from_cache(
        model.base, batch, cache,
        force_global_dropout=force_global_dropout,
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    if model._decoder_feature is None:
        raise RuntimeError('R9 cached path did not capture decoder feature.')
    coarse = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    if bool(force_global_dropout):
        coarse = torch.zeros_like(coarse)
        arterial_local = torch.zeros_like(arterial_local)
    center_logits, extent_logits, rescue_probability = model.rescue_head(
        batch['image'], model._decoder_feature, coarse, arterial_local
    )
    model._decoder_feature = None
    return {
        **base_output,
        'base_logits': base_output['logits'],
        'rescue_center_logits': center_logits,
        'rescue_extent_logits': extent_logits,
        'rescue_probability': rescue_probability,
        'arterial_global_map': cache['arterial_map'],
        'global_proposal_local': coarse,
    }


R9_VALIDATION_CAPTURE = []


@torch.no_grad()
def r9_infer_case(model, row, device):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(model.base, base_batch)

    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(
            native_shape[axis], LOCAL_PATCH_SIZE[axis],
            overlap=R9_VALIDATION_OVERLAP,
        )
        for axis in range(3)
    ]
    base_sum = np.zeros(native_shape, dtype=np.float32)
    rescue_sum = np.zeros(native_shape, dtype=np.float32)
    center_sum = np.zeros(native_shape, dtype=np.float32)
    extent_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][axis] + native_start[axis])
                    for axis in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = r9_cached_local_from_global(
                        model, batch, cache
                    )
                base_patch = torch.sigmoid(
                    output['base_logits'].float()
                )[0, 0].cpu().numpy()
                rescue_patch = output[
                    'rescue_probability'
                ].float()[0, 0].cpu().numpy()
                center_patch = torch.sigmoid(
                    output['rescue_center_logits'].float()
                )[0, 0].cpu().numpy()
                extent_patch = torch.sigmoid(
                    output['rescue_extent_logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[axis], min(
                        native_start[axis] + LOCAL_PATCH_SIZE[axis],
                        native_shape[axis],
                    ))
                    for axis in range(3)
                )
                copied_shape = tuple(
                    item.stop - item.start for item in destination
                )
                source = tuple(
                    slice(0, copied_shape[axis]) for axis in range(3)
                )
                weight = PILOT_WINDOW_WEIGHT[source]
                base_sum[destination] += base_patch[source] * weight
                rescue_sum[destination] += rescue_patch[source] * weight
                center_sum[destination] += center_patch[source] * weight
                extent_sum[destination] += extent_patch[source] * weight
                weight_sum[destination] += weight
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete R9 inference coverage."
        )
    base_probability = base_sum / weight_sum
    rescue_probability = rescue_sum / weight_sum
    center_probability = center_sum / weight_sum
    extent_probability = extent_sum / weight_sum
    probability = np.maximum(base_probability, rescue_probability)

    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(
            case['pad_start'][axis],
            case['pad_start'][axis] + native_shape[axis],
        )
        for axis in range(3)
    )
    proposal = proposal_canvas[native_slices]
    result = {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'base_probability': base_probability.astype(np.float16),
        'rescue_probability': rescue_probability.astype(np.float16),
        'center_probability': center_probability.astype(np.float16),
        'extent_probability': extent_probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'],
        'soft': case['soft'],
        'spacing': case['spacing'],
        'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }
    R9_VALIDATION_CAPTURE.append(result)
    return result


def r9_histogram_ap(cases, probability_key):
    positive_histogram = np.zeros(256, dtype=np.int64)
    negative_histogram = np.zeros(256, dtype=np.int64)
    for case in cases:
        row = case['row']
        if not as_bool(row['trial8_hard_metric_eligible']):
            continue
        probability = case[probability_key].astype(np.float32)
        target = case['target']
        positive_histogram += np.histogram(
            probability[target], bins=256, range=(0.0, 1.0)
        )[0]
        negative_histogram += np.histogram(
            probability[~target], bins=256, range=(0.0, 1.0)
        )[0]
    return float(pilot_histogram_average_precision(
        positive_histogram, negative_histogram
    ))


def r9_rescue_component_diagnostics(cases, threshold, epoch):
    rows = []
    positive_logits = []
    hard_negative_logits = []
    total_prediction = total_target = total_intersection = 0
    for case in cases:
        row = case['row']
        if not as_bool(row['trial8_hard_metric_eligible']):
            continue
        probability = case['rescue_probability'].astype(np.float32)
        target = np.asarray(case['target'], dtype=bool)
        prediction = probability >= float(threshold)
        total_prediction += int(prediction.sum())
        total_target += int(target.sum())
        total_intersection += int((prediction & target).sum())
        clipped = np.clip(probability, 1e-5, 1.0 - 1e-5)
        logits = np.log(clipped) - np.log1p(-clipped)
        if bool(target.any()):
            positive_logits.append(float(logits[target].mean()))
        background = logits[~target]
        if background.size:
            count = min(R9_CENTER_HARD_NEGATIVE_TOPK, background.size)
            hard_negative_logits.append(float(np.partition(
                background, background.size - count
            )[-count:].mean()))

        labels, _ = ndimage.label(
            target, np.ones((3, 3, 3), dtype=np.uint8)
        )
        predicted_distance = (
            ndimage.distance_transform_edt(
                ~prediction, sampling=case['spacing']
            ) if bool(prediction.any()) else None
        )
        iou_records = {
            int(item['component_id']): item
            for item in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            )
        }
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        for component_id, component_slice in enumerate(
            ndimage.find_objects(labels), start=1
        ):
            if component_slice is None:
                continue
            component = labels[component_slice] == component_id
            voxels = int(component.sum())
            tier = pilot_component_tier(
                voxels * voxel_volume_ml, voxels
            )
            maximum_probability = float(
                probability[component_slice][component].max(initial=0.0)
            )
            distance_mm = float('inf')
            if predicted_distance is not None:
                distance_mm = float(
                    predicted_distance[component_slice][component].min(
                        initial=np.inf
                    )
                )
            maximum_iou = float(
                iou_records[component_id]['maximum_component_iou']
            )
            rows.append({
                'epoch': int(epoch),
                'subject_id': case['subject_id'],
                'component_id': int(component_id),
                'component_tier': tier,
                'component_voxels': voxels,
                'rescue_probability_maximum': maximum_probability,
                'rescue_maximum_component_iou': maximum_iou,
                'rescue_detected': maximum_iou >= 0.10,
                'within_2mm': distance_mm <= 2.0,
                'within_4mm': distance_mm <= 4.0,
                'within_8mm': distance_mm <= 8.0,
                'minimum_distance_mm': distance_mm,
                'representation_failure_below_0p10': (
                    maximum_probability < 0.10
                ),
            })
    frame = pd.DataFrame(rows)
    small = (
        frame[frame['component_tier'].astype(str).isin(R9_SMALL_TIERS)]
        if len(frame) else frame
    )
    positive_mean = float(np.mean(positive_logits)) if positive_logits else 0.0
    negative_mean = (
        float(np.mean(hard_negative_logits))
        if hard_negative_logits else 0.0
    )
    return frame, {
        'rescue_component_recall': (
            float(frame['rescue_detected'].mean()) if len(frame) else 0.0
        ),
        'rescue_small_component_recall': (
            float(small['rescue_detected'].mean()) if len(small) else 0.0
        ),
        'rescue_small_within_2mm_fraction': (
            float(small['within_2mm'].mean()) if len(small) else 0.0
        ),
        'rescue_small_within_4mm_fraction': (
            float(small['within_4mm'].mean()) if len(small) else 0.0
        ),
        'rescue_small_within_8mm_fraction': (
            float(small['within_8mm'].mean()) if len(small) else 0.0
        ),
        'rescue_small_representation_failure_fraction': (
            float(small['representation_failure_below_0p10'].mean())
            if len(small) else 0.0
        ),
        'rescue_positive_logit_mean': positive_mean,
        'rescue_hard_negative_logit_mean': negative_mean,
        'rescue_positive_minus_hard_negative_margin': (
            positive_mean - negative_mean
        ),
        'rescue_predicted_to_target_volume_ratio': float(
            total_prediction / max(total_target, 1)
        ),
        'rescue_voxel_precision': float(
            total_intersection / max(total_prediction, 1)
        ),
    }


@torch.no_grad()
def r9_validate(model, epoch, device):
    global PILOT_VALIDATION_ROOT, LOCAL_THRESHOLDS, PROPOSAL_THRESHOLDS
    global pilot_infer_case, R9_VALIDATION_CAPTURE
    original_root = PILOT_VALIDATION_ROOT
    original_local = LOCAL_THRESHOLDS
    original_proposal = PROPOSAL_THRESHOLDS
    original_infer = pilot_infer_case
    R9_VALIDATION_CAPTURE = []
    try:
        PILOT_VALIDATION_ROOT = R9_VALIDATION_ROOT
        LOCAL_THRESHOLDS = R9_LOCAL_THRESHOLDS
        PROPOSAL_THRESHOLDS = R9_PROPOSAL_THRESHOLDS
        pilot_infer_case = r9_infer_case
        summary, tier_rows = pilot_validate(model, epoch, device)
    finally:
        PILOT_VALIDATION_ROOT = original_root
        LOCAL_THRESHOLDS = original_local
        PROPOSAL_THRESHOLDS = original_proposal
        pilot_infer_case = original_infer

    cases = list(R9_VALIDATION_CAPTURE)
    if len(cases) != len(heldout_rows):
        raise RuntimeError('R9 validation capture is incomplete.')
    rescue_threshold_frame = pd.DataFrame([
        pilot_threshold_metrics(cases, threshold, 'rescue_probability')
        for threshold in R9_RESCUE_THRESHOLDS
    ])
    rescue_threshold_frame['rescue_selection_score'] = (
        0.40 * rescue_threshold_frame['small_component_recall']
        + 0.20 * rescue_threshold_frame['component_recall']
        + 0.15 * rescue_threshold_frame['voxel_precision']
        + 0.15 * rescue_threshold_frame['lesion_f1']
        + 0.10 * rescue_threshold_frame['small_case_mean_dice']
    )
    rescue_threshold_frame = rescue_threshold_frame.sort_values(
        [
            'rescue_selection_score', 'small_component_recall',
            'voxel_precision', 'threshold',
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    rescue_selected = rescue_threshold_frame.iloc[0].to_dict()
    rescue_threshold = float(rescue_selected['threshold'])
    component_frame, component_summary = (
        r9_rescue_component_diagnostics(
            cases, rescue_threshold, epoch
        )
    )
    rescue_ap = r9_histogram_ap(cases, 'rescue_probability')

    case_rows = []
    base_to_final_target = []
    base_to_final_background = []
    for case in cases:
        base = case['base_probability'].astype(np.float32)
        final = case['probability'].astype(np.float32)
        rescue = case['rescue_probability'].astype(np.float32)
        target = case['target']
        case_rows.append({
            'epoch': int(epoch),
            'subject_id': case['subject_id'],
            'rescue_probability_mean': float(rescue.mean()),
            'rescue_probability_maximum': float(rescue.max(initial=0.0)),
            'base_to_final_mean_absolute_delta': float(
                np.abs(final - base).mean()
            ),
            'base_to_final_target_mean_absolute_delta': (
                float(np.abs(final[target] - base[target]).mean())
                if bool(target.any()) else float('nan')
            ),
            'base_to_final_background_mean_absolute_delta': float(
                np.abs(final[~target] - base[~target]).mean()
            ),
        })
        if bool(target.any()):
            base_to_final_target.append(float(
                np.abs(final[target] - base[target]).mean()
            ))
        base_to_final_background.append(float(
            np.abs(final[~target] - base[~target]).mean()
        ))

    summary.update({
        'rescue_threshold': rescue_threshold,
        'rescue_voxel_average_precision_256bin': rescue_ap,
        'rescue_threshold_small_component_recall': float(
            rescue_selected['small_component_recall']
        ),
        'rescue_threshold_component_recall': float(
            rescue_selected['component_recall']
        ),
        'rescue_threshold_voxel_precision': float(
            rescue_selected['voxel_precision']
        ),
        'rescue_threshold_lesion_f1': float(
            rescue_selected['lesion_f1']
        ),
        'base_to_final_target_mean_absolute_delta': (
            float(np.mean(base_to_final_target))
            if base_to_final_target else 0.0
        ),
        'base_to_final_background_mean_absolute_delta': (
            float(np.mean(base_to_final_background))
            if base_to_final_background else 0.0
        ),
        **component_summary,
    })
    required = [
        'positive_mean_dice', 'positive_global_dice',
        'small_case_mean_dice', 'small_component_recall',
        'rescue_voxel_average_precision_256bin',
        'rescue_small_component_recall',
        'rescue_positive_minus_hard_negative_margin',
    ]
    if not all(np.isfinite(float(summary[name])) for name in required):
        raise RuntimeError('R9 validation produced non-finite metrics.')
    epoch_root = R9_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    atomic_write_csv(
        rescue_threshold_frame,
        epoch_root / 'rescue_threshold_summary.csv',
    )
    atomic_write_csv(
        pd.DataFrame(case_rows),
        epoch_root / 'rescue_case_metrics.csv',
    )
    atomic_write_csv(
        component_frame,
        epoch_root / 'rescue_component_metrics.csv',
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del cases
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows


def r9_baseline_metrics():
    baseline_record = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
    summary = dict(baseline_record['validation_summary'])
    return {
        'r5_mean_dice': float(summary['positive_mean_dice']),
        'r5_global_dice': float(summary['positive_global_dice']),
        'r5_small_case_dice': r8p_r5_small_case_dice(summary),
        'r5_small_component_recall': r8p_r5_small_component_recall(
            baseline_record
        ),
        'mean_dice_floor': float(summary['positive_mean_dice']) - 0.005,
        'global_dice_floor': float(summary['positive_global_dice']) - 0.010,
        'small_case_dice_floor': max(
            0.0, r8p_r5_small_case_dice(summary) - 0.005
        ),
    }


def r9_checkpoint_evaluation(summary, baseline):
    metrics = {
        'positive_mean_dice': float(summary['positive_mean_dice']),
        'positive_global_dice': float(summary['positive_global_dice']),
        'small_case_mean_dice': float(summary['small_case_mean_dice']),
        'small_component_recall': float(summary['small_component_recall']),
        'lesion_f1': float(summary['lesion_f1']),
        'rescue_ap': float(summary['rescue_voxel_average_precision_256bin']),
    }
    guards = {
        'mean_dice_floor': (
            metrics['positive_mean_dice'] >= baseline['mean_dice_floor']
        ),
        'global_dice_floor': (
            metrics['positive_global_dice'] >= baseline['global_dice_floor']
        ),
        'small_case_dice_floor': (
            metrics['small_case_mean_dice']
            >= baseline['small_case_dice_floor']
        ),
        'small_component_improved': (
            metrics['small_component_recall']
            > baseline['r5_small_component_recall']
        ),
    }
    selection_key = (
        metrics['small_component_recall'],
        metrics['positive_mean_dice'],
        metrics['small_case_mean_dice'],
        metrics['positive_global_dice'],
        metrics['lesion_f1'],
        metrics['rescue_ap'],
    )
    diagnostic_score = (
        0.35 * metrics['small_component_recall']
        + 0.20 * float(summary['rescue_small_component_recall'])
        + 0.20 * metrics['positive_mean_dice']
        + 0.10 * metrics['small_case_mean_dice']
        + 0.10 * metrics['rescue_ap']
        + 0.05 * metrics['positive_global_dice']
    )
    return (
        all(guards.values()), guards, selection_key,
        float(diagnostic_score), metrics,
    )


def r9_existing_preflight_complete():
    if not R9_PREFLIGHT_GATE_PATH.exists():
        return False
    gate = load_json(R9_PREFLIGHT_GATE_PATH)
    if not (
        gate.get('preflight_passed') is True
        and gate.get('model_version') == R9_MODEL_VERSION
        and gate.get('preflight_version') == R9_PREFLIGHT_VERSION
        and gate.get('code_revision') == R9_CODE_REVISION
        and gate.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
    ):
        return False
    hashes = gate.get('artifact_sha256', {})
    return all(
        path.exists() and hashes.get(path.name) == file_sha256(path)
        for path in R9_PREFLIGHT_OUTPUTS
    )


def run_r9_preflight():
    if r9_existing_preflight_complete():
        gate = load_json(R9_PREFLIGHT_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R9 COMPONENT-RESCUE PREFLIGHT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Total parameters: {int(gate["total_parameters"]):,}')
        print(f'Preflight gate:   {R9_PREFLIGHT_GATE_PATH}')
        print('No source checkpoint or preflight artifact was rewritten.')
        return gate

    R9_PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_sha_before = file_sha256(R5_BEST_CHECKPOINT_PATH)
    r8_history_rows = 0
    r8_last_epoch = 0
    if R8P_HISTORY_PATH.exists():
        r8_history = pd.read_csv(R8P_HISTORY_PATH)
        r8_history_rows = int(len(r8_history))
        if len(r8_history):
            r8_last_epoch = int(r8_history['epoch'].max())
    upstream_checks = {
        'r5_pilot_complete': r5_result.get('pilot_complete') is True,
        'r5_checkpoint_exists': R5_BEST_CHECKPOINT_PATH.exists(),
        'r5_checkpoint_hash_matches_result': (
            r5_result.get('best_checkpoint_sha256') == r5_sha_before
        ),
        'r8_architecture_preflight_passed': (
            load_json(R8_GATE_PATH).get('preflight_passed') is True
        ),
        'r8_weights_are_not_an_r9_source': True,
        'r7_weights_are_not_an_r9_source': True,
        'fold0_partition_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_and_test_counts_match': (
            len(locked_validation_df) == 23 and len(sealed_test_df) == 23
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R9 UPSTREAM PREFLIGHT FAILED. Do not train. Failed: '
            + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R9 FROZEN-R5 COMPONENT-RESCUE PREFLIGHT STARTING')
    print('=' * 80)
    print('Weight source:                       R5 best EMA only')
    print('R7 / R8 weights used:               False / False')
    print(f'Observed partial R8 history:         {r8_history_rows} rows, '
          f'last epoch {r8_last_epoch}')
    print('Target access:                       fold-0 fit smoke cases only')
    print('Fold-0 held-out targets read:        False')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Optimizer steps:                     0')

    if not torch.cuda.is_available():
        raise RuntimeError('R9 preflight requires a CUDA runtime.')
    device = torch.device('cuda')
    set_determinism(R9_SEED)
    random.seed(R9_SEED)
    checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'r5_checkpoint_pilot_version_matches': (
            checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'r5_checkpoint_code_revision_matches': (
            checkpoint.get('code_revision') == R5_PILOT_CODE_REVISION
        ),
        'r5_checkpoint_parameter_count_matches': (
            int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'r5_checkpoint_contains_ema_state': (
            'ema_model_state' in checkpoint
        ),
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 checkpoint for R9: ' + ', '.join(failed))
    r5_state = checkpoint['ema_model_state']

    dataset = Trial8R9Fold0Dataset()
    probe_indices = []
    for tier in R8_TIER_QUOTAS:
        probe_indices.append(next(
            index for index, entry in enumerate(dataset.schedule)
            if str(entry['sample_type']) == 'positive'
            and str(entry['component_tier']) == tier
        ))
    probe_indices.append(next(
        index for index, entry in enumerate(dataset.schedule)
        if str(entry['sample_type']) != 'positive'
    ))
    target_rows = []
    for index in probe_indices:
        sample = dataset[index]
        center = np.asarray(sample['rescue_center_target'])
        extent = np.asarray(sample['rescue_extent_target'])
        valid = np.asarray(sample['rescue_supervision_valid'])
        positive = bool(np.asarray(
            sample['rescue_component_positive']
        ).item())
        target_rows.append({
            'subject_id': sample['subject_id'],
            'sample_type': sample['sample_type'],
            'component_tier': sample['component_tier'],
            'component_id': int(np.asarray(
                sample['atomic_component_id']
            ).item()),
            'positive_case': positive,
            'exact_voxels': int((extent > 0.5).sum()),
            'center_support_voxels': int((center > 0.05).sum()),
            'valid_supervision_voxels': int(valid.sum()),
            'ignored_other_component_voxels': int(np.asarray(
                sample['ignored_other_component_voxels']
            ).item()),
            'center_maximum': float(center.max(initial=0.0)),
            'center_minimum': float(center.min(initial=0.0)),
            'exact_is_inside_center_maximum': bool(
                np.all(center[extent > 0.5] == 1.0)
            ) if bool((extent > 0.5).any()) else True,
            'exact_is_supervised': bool(
                np.all(valid[extent > 0.5])
            ) if bool((extent > 0.5).any()) else True,
            'all_targets_finite': bool(
                np.isfinite(center).all() and np.isfinite(extent).all()
            ),
        })
    target_frame = pd.DataFrame(target_rows)
    atomic_write_csv(target_frame, R9_TARGET_AUDIT_PATH)

    positive_index = probe_indices[0]
    negative_index = probe_indices[-1]
    positive_batch = torch.utils.data.default_collate([
        dataset[positive_index]
    ])
    negative_batch = torch.utils.data.default_collate([
        dataset[negative_index]
    ])

    reference = VascuFateNetV22(pilot_schema).to(device).eval()
    reference.load_state_dict(r5_state, strict=True)
    for parameter in reference.parameters():
        parameter.requires_grad_(False)
    positive_device = pilot_to_device(positive_batch, device)
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        reference_output = reference(
            **{key: positive_device[key] for key in MODEL_INPUT_KEYS_V22},
            **R9_NO_DROPOUT,
        )
    reference_logits = reference_output['logits'].detach().float().cpu()
    del reference, reference_output
    gc.collect()
    torch.cuda.empty_cache()

    model = VascuFateNetR9(pilot_schema).to(device)
    model.load_r5_base_state(r5_state)
    total_parameters = parameter_count(model)
    base_parameters = parameter_count(model.base)
    rescue_parameters = parameter_count(model.rescue_head)
    trainable_parameters = parameter_count(model, trainable_only=True)
    parameter_frame = pd.DataFrame([
        {'component': 'frozen_r5_base', 'parameters': base_parameters,
         'trainable': False},
        {'component': 'center_extent_rescue_head',
         'parameters': rescue_parameters, 'trainable': True},
        {'component': 'total', 'parameters': total_parameters,
         'trainable': True},
    ])
    atomic_write_csv(parameter_frame, R9_PARAMETER_AUDIT_PATH)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model.eval()
    with torch.no_grad(), torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        output = model(
            **{key: positive_device[key] for key in MODEL_INPUT_KEYS_V22},
            **R9_NO_DROPOUT,
        )
    peak_eval_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    base_difference = float(torch.max(torch.abs(
        output['logits'].detach().float().cpu() - reference_logits
    )))
    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'base_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'rescue_center_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'rescue_extent_logits': (1, 1, *LOCAL_PATCH_SIZE),
        'rescue_probability': (1, 1, *LOCAL_PATCH_SIZE),
    }
    output_rows = []
    for name, expected in expected_shapes.items():
        observed = tuple(output[name].shape)
        output_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected),
            'shape_matches': observed == expected,
            'finite': bool(torch.isfinite(output[name]).all().item()),
        })
    output_frame = pd.DataFrame(output_rows)
    atomic_write_csv(output_frame, R9_OUTPUT_AUDIT_PATH)

    state_digest_before = r9_model_digest(model)
    frozen_base_digest_before = r9_model_digest(model.base)
    model.train()
    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        positive_loss, positive_rows, positive_output = r9_training_loss(
            model, positive_batch, device
        )
        negative_loss, negative_rows, negative_output = r9_training_loss(
            model, negative_batch, device
        )
        smoke_loss = 0.5 * positive_loss + 0.5 * negative_loss
    smoke_loss.backward()
    peak_gradient_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    rescue_norm, rescue_gradient_observed, rescue_gradient_finite = (
        r8_gradient_norm(model.rescue_head)
    )
    base_norm, base_gradient_observed, base_gradient_finite = (
        r8_gradient_norm(model.base)
    )
    gradient_frame = pd.DataFrame([
        {
            'component': 'frozen_r5_base',
            'gradient_norm': base_norm,
            'gradient_observed': base_gradient_observed,
            'gradient_finite': base_gradient_finite,
        },
        {
            'component': 'center_extent_rescue_head',
            'gradient_norm': rescue_norm,
            'gradient_observed': rescue_gradient_observed,
            'gradient_finite': rescue_gradient_finite,
        },
    ])
    atomic_write_csv(gradient_frame, R9_GRADIENT_AUDIT_PATH)
    state_digest_after = r9_model_digest(model)
    frozen_base_digest_after = r9_model_digest(model.base)
    loss_frame = pd.DataFrame([
        {
            'sample_type': 'positive',
            'loss': float(positive_loss.detach().float().cpu()),
            **positive_rows[0],
        },
        {
            'sample_type': 'negative',
            'loss': float(negative_loss.detach().float().cpu()),
            **negative_rows[0],
        },
        {
            'sample_type': 'combined_smoke',
            'loss': float(smoke_loss.detach().float().cpu()),
        },
    ])
    atomic_write_csv(loss_frame, R9_LOSS_AUDIT_PATH)

    policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R9_MODEL_VERSION,
        'preflight_version': R9_PREFLIGHT_VERSION,
        'code_revision': R9_CODE_REVISION,
        'warm_start': 'completed_r5_best_ema_only',
        'r7_weights_used': False,
        'r8_weights_used': False,
        'frozen_r5_base_for_all_epochs': True,
        'architecture': {
            'raw_image_stem_channels': 8,
            'frozen_decoder_and_context_channels': 12,
            'fused_channels': 24,
            'outputs': ['dense_center_halo', 'exact_component_extent'],
            'final_probability': 'maximum_of_r5_and_rescue_probability',
            'scalar_gate': False,
            'direct_graph_to_logit_residual': False,
        },
        'target_contract': {
            'center_sigma_mm': R9_CENTER_SIGMA_MM,
            'center_radius_mm': R9_CENTER_RADIUS_MM,
            'other_component_ignore_mm': R9_OTHER_COMPONENT_IGNORE_MM,
            'other_true_components_are_negatives': False,
        },
        'optimization': {
            'maximum_epochs': R9_MAX_EPOCHS,
            'minimum_epochs': R9_MINIMUM_EPOCHS,
            'early_stopping_patience': R9_EARLY_STOPPING_PATIENCE,
            'early_stopping_minimum_delta': (
                R9_EARLY_STOPPING_MINIMUM_DELTA
            ),
            'peak_learning_rate': R9_PEAK_LR,
            'weight_decay': R9_WEIGHT_DECAY,
            'gradient_clip_norm': R9_GRADIENT_CLIP_NORM,
            'validation_every_epoch': True,
            'trainable_modules': ['center_extent_rescue_head'],
        },
        'checkpoint_guards': {
            'r5_mean_dice_minus': 0.005,
            'r5_global_dice_minus': 0.010,
            'r5_small_case_dice_minus': 0.005,
            'small_component_recall_must_improve': True,
            'oracle_metrics_used_for_selection': False,
        },
        'fold0_heldout_targets_for_training': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    atomic_write_json(policy, R9_POLICY_PATH)

    r5_sha_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    checks = {
        **upstream_checks,
        **checkpoint_checks,
        'schedule_has_168_samples': len(dataset) == 168,
        'target_probe_covers_four_positive_tiers_and_negative': (
            len(target_frame) == 5
            and int(target_frame['positive_case'].sum()) == 4
        ),
        'positive_exact_targets_are_nonempty': bool(
            (target_frame[target_frame['positive_case']][
                'exact_voxels'
            ] > 0).all()
        ),
        'positive_center_targets_are_denser_than_exact_targets': bool(
            (target_frame[target_frame['positive_case']][
                'center_support_voxels'
            ] > target_frame[target_frame['positive_case']][
                'exact_voxels'
            ]).all()
        ),
        'negative_targets_are_empty': bool(
            (target_frame[~target_frame['positive_case']][
                'exact_voxels'
            ] == 0).all()
        ),
        'all_exact_targets_are_supervised': bool(
            target_frame['exact_is_supervised'].all()
        ),
        'all_target_arrays_are_finite': bool(
            target_frame['all_targets_finite'].all()
        ),
        'base_parameter_count_matches_r5': (
            base_parameters == EXPECTED_V22_PARAMETERS
        ),
        'rescue_head_is_only_trainable_module': (
            trainable_parameters == rescue_parameters
            and rescue_parameters > 0
        ),
        'total_parameter_count_is_additive': (
            total_parameters == base_parameters + rescue_parameters
        ),
        'r5_forward_is_exactly_preserved_before_union': (
            base_difference <= 1e-6
        ),
        'all_output_shapes_match': bool(output_frame['shape_matches'].all()),
        'all_outputs_are_finite': bool(output_frame['finite'].all()),
        'rescue_probability_is_bounded': bool(
            (output['rescue_probability'] >= 0).all().item()
            and (output['rescue_probability'] <= 1).all().item()
        ),
        'positive_smoke_loss_is_finite': bool(
            torch.isfinite(positive_loss).item()
        ),
        'negative_smoke_loss_is_finite': bool(
            torch.isfinite(negative_loss).item()
        ),
        'rescue_head_has_finite_nonzero_gradient': (
            rescue_gradient_observed and rescue_gradient_finite
            and rescue_norm > 0.0
        ),
        'frozen_r5_base_has_no_gradient': (
            not base_gradient_observed and base_norm == 0.0
        ),
        'frozen_r5_base_state_unchanged_by_backward': (
            frozen_base_digest_before == frozen_base_digest_after
        ),
        'model_state_unchanged_by_backward_without_step': (
            state_digest_before == state_digest_after
        ),
        'peak_eval_memory_below_20gb': peak_eval_memory_gb < 20.0,
        'peak_gradient_memory_below_20gb': peak_gradient_memory_gb < 20.0,
        'r5_checkpoint_unchanged': r5_sha_before == r5_sha_after,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'fold0_heldout_targets_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'full_fivefold_training_remains_locked': True,
    }
    checks_frame = pd.DataFrame([
        {'check': name, 'passed': bool(value)}
        for name, value in checks.items()
    ])
    atomic_write_csv(checks_frame, R9_CHECKS_PATH)
    preflight_passed = all(checks.values())
    artifact_hashes = {
        path.name: file_sha256(path) for path in R9_PREFLIGHT_OUTPUTS
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': R9_MODEL_VERSION,
        'preflight_version': R9_PREFLIGHT_VERSION,
        'code_revision': R9_CODE_REVISION,
        'preflight_passed': bool(preflight_passed),
        'r5_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'r5_checkpoint_sha256': r5_sha_before,
        'r7_weights_used': False,
        'r8_weights_used': False,
        'base_parameters': base_parameters,
        'rescue_parameters': rescue_parameters,
        'total_parameters': total_parameters,
        'peak_eval_memory_gb': peak_eval_memory_gb,
        'peak_gradient_memory_gb': peak_gradient_memory_gb,
        'optimizer_steps': 0,
        'training_performed': False,
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'r9_pilot_training_authorized': bool(preflight_passed),
        'full_fivefold_training_authorized': False,
        'checks': checks,
        'artifact_sha256': artifact_hashes,
        'policy_path': str(R9_POLICY_PATH),
    }
    atomic_write_json(gate, R9_PREFLIGHT_GATE_PATH)
    model.zero_grad(set_to_none=True)
    del model, positive_batch, negative_batch, positive_device
    del positive_output, negative_output, output, checkpoint
    gc.collect()
    torch.cuda.empty_cache()
    if not preflight_passed:
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R9 COMPONENT-RESCUE PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )

    print('\nR9 target geometry audit:')
    display(target_frame)
    print('\nR9 parameter audit:')
    display(parameter_frame)
    print('\nR9 gradient audit:')
    display(gradient_frame)
    print('\nR9 preflight checks:')
    display(checks_frame)
    print('\n' + '=' * 80)
    print('TRIAL8 R9 FROZEN-R5 COMPONENT-RESCUE PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Frozen R5 parameters:               {base_parameters:,}')
    print(f'New rescue parameters:              {rescue_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak gradient memory GB:            {peak_gradient_memory_gb:.3f}')
    print('Other true components as negatives: False')
    print('Scalar rescue gate:                 False')
    print('R5 gradient path:                   blocked')
    print('Fold-0 held-out targets read:       False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer steps:                    0')
    print('R9 pilot training authorized:       True')
    print('Full five-fold training authorized: False')
    print(f'Preflight gate:                     {R9_PREFLIGHT_GATE_PATH}')
    return gate


def r9_existing_result_complete():
    if not R9_RESULT_GATE_PATH.exists():
        return False
    result = load_json(R9_RESULT_GATE_PATH)
    if not (
        result.get('pilot_complete') is True
        and result.get('pilot_version') == R9_PILOT_VERSION
        and result.get('code_revision') == R9_CODE_REVISION
        and result.get('preflight_gate_sha256')
        == file_sha256(R9_PREFLIGHT_GATE_PATH)
        and result.get('r5_checkpoint_sha256')
        == file_sha256(R5_BEST_CHECKPOINT_PATH)
        and R9_LAST_CHECKPOINT_PATH.exists()
        and result.get('last_checkpoint_sha256')
        == file_sha256(R9_LAST_CHECKPOINT_PATH)
        and R9_HISTORY_PATH.exists()
        and R9_FINAL_SUMMARY_PATH.exists()
        and result.get('final_summary_sha256')
        == file_sha256(R9_FINAL_SUMMARY_PATH)
    ):
        return False
    if bool(result.get('eligible_r9_checkpoint_found')):
        return (
            R9_BEST_CHECKPOINT_PATH.exists()
            and result.get('best_checkpoint_sha256')
            == file_sha256(R9_BEST_CHECKPOINT_PATH)
        )
    return not R9_BEST_CHECKPOINT_PATH.exists()


def r9_ema_update(ema_model, model, optimizer_step):
    decay = min(
        EMA_MAX_DECAY,
        1.0 - 1.0 / float(int(optimizer_step) + 1),
    )
    with torch.no_grad():
        source = model.state_dict()
        destination = ema_model.state_dict()
        if source.keys() != destination.keys():
            raise RuntimeError('R9 EMA state schemas differ.')
        for name, value in destination.items():
            source_value = source[name].detach()
            
            
            
            if name.startswith('base.'):
                value.copy_(source_value)
            elif value.dtype.is_floating_point:
                value.mul_(decay).add_(source_value, alpha=1.0 - decay)
            else:
                value.copy_(source_value)


def run_r9_fold0_pilot():
    if r9_existing_result_complete():
        result = load_json(R9_RESULT_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R9 COMPONENT-RESCUE FOLD-0 PILOT ALREADY COMPLETE')
        print('=' * 80)
        print(f'Eligible checkpoint: {result.get("eligible_r9_checkpoint_found")}')
        print(f'Best epoch:          {result.get("best_epoch")}')
        print(f'Result gate:         {R9_RESULT_GATE_PATH}')
        print('No checkpoint or training artifact was rewritten.')
        return

    preflight_gate = load_json(R9_PREFLIGHT_GATE_PATH)
    policy = load_json(R9_POLICY_PATH)
    r5_result = load_json(R5_RESULT_GATE_PATH)
    r5_checkpoint_sha256 = file_sha256(R5_BEST_CHECKPOINT_PATH)
    baseline = r9_baseline_metrics()
    upstream_checks = {
        'r9_preflight_passed': (
            preflight_gate.get('preflight_passed') is True
        ),
        'r9_pilot_training_authorized': (
            preflight_gate.get('r9_pilot_training_authorized') is True
        ),
        'fivefold_training_locked': (
            preflight_gate.get('full_fivefold_training_authorized') is False
        ),
        'preflight_model_version_matches': (
            preflight_gate.get('model_version') == R9_MODEL_VERSION
        ),
        'preflight_revision_matches': (
            preflight_gate.get('code_revision') == R9_CODE_REVISION
        ),
        'r5_checkpoint_hash_matches': (
            preflight_gate.get('r5_checkpoint_sha256')
            == r5_checkpoint_sha256
        ),
        'r7_weights_excluded': (
            preflight_gate.get('r7_weights_used') is False
        ),
        'r8_weights_excluded': (
            preflight_gate.get('r8_weights_used') is False
        ),
        'policy_freezes_r5': (
            policy.get('frozen_r5_base_for_all_epochs') is True
        ),
        'fold0_partition_counts_match': (
            len(fold0_fit_df) == 82 and len(fold0_heldout_df) == 20
        ),
        'locked_validation_and_test_counts_match': (
            len(locked_validation_df) == 23 and len(sealed_test_df) == 23
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R9 TRAINING UPSTREAM GATE FAILED. Do not train. '
            'Failed: ' + ', '.join(failed)
        )
    if R9_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            'An incompatible R9 result gate exists. Refusing overwrite.'
        )
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 R9 fold-0 pilot requires CUDA.')

    R9_PILOT_ROOT.mkdir(parents=True, exist_ok=True)
    R9_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
    R9_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
    random.seed(R9_SEED)
    set_determinism(R9_SEED)
    device = torch.device('cuda')
    r5_checkpoint = torch.load(
        R5_BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False
    )
    checkpoint_checks = {
        'r5_version_matches': (
            r5_checkpoint.get('pilot_version') == R5_PILOT_VERSION
        ),
        'r5_revision_matches': (
            r5_checkpoint.get('code_revision')
            == R5_PILOT_CODE_REVISION
        ),
        'r5_parameter_count_matches': (
            int(r5_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'r5_ema_state_exists': 'ema_model_state' in r5_checkpoint,
    }
    if not all(checkpoint_checks.values()):
        failed = [name for name, passed in checkpoint_checks.items() if not passed]
        raise RuntimeError('Unsafe R5 warm start for R9: ' + ', '.join(failed))
    r5_state = {
        name: value.detach().cpu().clone()
        for name, value in r5_checkpoint['ema_model_state'].items()
    }
    del r5_checkpoint

    model = VascuFateNetR9(pilot_schema).to(device)
    model.load_r5_base_state(r5_state)
    expected_parameters = int(preflight_gate['total_parameters'])
    if parameter_count(model) != expected_parameters:
        raise RuntimeError('R9 parameter count changed after preflight.')
    if parameter_count(model, trainable_only=True) != int(
        preflight_gate['rescue_parameters']
    ):
        raise RuntimeError('R9 trainable parameter count changed.')
    frozen_base_digest = r9_model_digest(model.base)

    rng_before_ema = pilot_capture_rng_state()
    ema_model = VascuFateNetR9(pilot_schema).to(device).eval()
    ema_model.load_state_dict(model.state_dict(), strict=True)
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    pilot_restore_rng_state(rng_before_ema)

    optimizer = torch.optim.AdamW(
        model.rescue_head.parameters(), lr=R9_PEAK_LR / 2.0,
        betas=(0.9, 0.99), eps=1e-8,
        weight_decay=R9_WEIGHT_DECAY,
    )
    dataset = Trial8R9Fold0Dataset()
    expected_updates_per_epoch = int(math.ceil(
        len(dataset) / R9_ACCUMULATION_STEPS
    ))
    if len(dataset) != 168 or expected_updates_per_epoch != 42:
        raise RuntimeError('R9 epoch/update cardinality changed.')

    config = {
        'pilot_version': R9_PILOT_VERSION,
        'code_revision': R9_CODE_REVISION,
        'model_version': R9_MODEL_VERSION,
        'fold': FOLD0,
        'fit_subjects': 82,
        'heldout_subjects': 20,
        'parameters': expected_parameters,
        'trainable_parameters': int(preflight_gate['rescue_parameters']),
        'samples_per_epoch': len(dataset),
        'optimizer_updates_per_epoch': expected_updates_per_epoch,
        'maximum_epochs': R9_MAX_EPOCHS,
        'minimum_epochs': R9_MINIMUM_EPOCHS,
        'early_stopping_patience': R9_EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': R9_EARLY_STOPPING_MINIMUM_DELTA,
        'validation_every_epoch': True,
        'learning_rate_policy': {
            str(epoch): r9_optimizer_lr(epoch)
            for epoch in range(1, R9_MAX_EPOCHS + 1)
        },
        'checkpoint_baseline': baseline,
        'warm_start': 'r5_best_ema_only',
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'r7_weights_used': False,
        'r8_weights_used': False,
        'r5_frozen_for_all_epochs': True,
        'preflight_gate_sha256': file_sha256(R9_PREFLIGHT_GATE_PATH),
        'policy_sha256': file_sha256(R9_POLICY_PATH),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
    }
    config_hash = pilot_config_hash(config)
    if R9_CONFIG_PATH.exists():
        existing = load_json(R9_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing R9 pilot configuration differs. Refusing overwrite.'
            )
    else:
        atomic_write_json(config, R9_CONFIG_PATH)
    if pilot_config_hash(load_json(R9_CONFIG_PATH)) != config_hash:
        raise RuntimeError('R9 pilot configuration round-trip is unstable.')

    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': R9_PILOT_VERSION,
        'code_revision': R9_CODE_REVISION,
        'config_hash': config_hash,
        'preflight_gate_sha256': file_sha256(R9_PREFLIGHT_GATE_PATH),
        'policy_sha256': file_sha256(R9_POLICY_PATH),
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'r5_best_ema_used_for_initialization': True,
        'r5_frozen_for_all_epochs': True,
        'r7_weights_used': False,
        'r8_weights_used': False,
        'same_r9_last_checkpoint_only_resume': True,
        'r9_fold0_pilot_training_authorized': True,
        'full_fivefold_training_authorized': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'upstream_checks': upstream_checks,
    }
    if R9_START_GATE_PATH.exists():
        existing = load_json(R9_START_GATE_PATH)
        immutable = [
            'pilot_version', 'code_revision', 'config_hash',
            'preflight_gate_sha256', 'policy_sha256',
            'r5_checkpoint_sha256',
        ]
        if any(existing.get(name) != start_gate.get(name) for name in immutable):
            raise RuntimeError('Existing R9 start gate differs.')
    else:
        atomic_write_json(start_gate, R9_START_GATE_PATH)

    start_epoch = 1
    optimizer_steps = 0
    best_epoch = 0
    best_selection_key = None
    best_metrics = None
    diagnostic_best_epoch = 0
    diagnostic_best_score = -float('inf')
    monitor_reference = -float('inf')
    validations_without_improvement = 0
    history_rows = []
    resumed = False
    if R9_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            R9_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': (
                checkpoint.get('pilot_version') == R9_PILOT_VERSION
            ),
            'code_revision': (
                checkpoint.get('code_revision') == R9_CODE_REVISION
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'r5_checkpoint_sha256': (
                checkpoint.get('r5_checkpoint_sha256')
                == r5_checkpoint_sha256
            ),
            'parameter_count': (
                int(checkpoint.get('parameter_count', -1))
                == expected_parameters
            ),
            'r5_frozen': checkpoint.get('r5_frozen') is True,
            'locked_validation_targets_not_read': (
                checkpoint.get('locked_validation_targets_read') is False
            ),
            'test_targets_not_read': (
                checkpoint.get('test_targets_read') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [name for name, passed in resume_checks.items() if not passed]
            raise RuntimeError('Unsafe R9 resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'], strict=True)
        ema_model.load_state_dict(checkpoint['ema_model_state'], strict=True)
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        optimizer_steps = int(checkpoint['optimizer_steps'])
        best_epoch = int(checkpoint.get('best_epoch', 0))
        best_selection_key = checkpoint.get('best_selection_key')
        best_metrics = checkpoint.get('best_metrics')
        diagnostic_best_epoch = int(
            checkpoint.get('diagnostic_best_epoch', 0)
        )
        diagnostic_best_score = float(
            checkpoint.get('diagnostic_best_score', -float('inf'))
        )
        monitor_reference = float(
            checkpoint.get('monitor_reference', -float('inf'))
        )
        validations_without_improvement = int(
            checkpoint.get('validations_without_improvement', 0)
        )
        if not R9_HISTORY_PATH.exists():
            raise RuntimeError('R9 resume checkpoint exists without history.')
        history_rows = pd.read_csv(R9_HISTORY_PATH).to_dict('records')
        if max(int(row['epoch']) for row in history_rows) != start_epoch - 1:
            raise RuntimeError('R9 resume history/checkpoint epochs disagree.')
        if optimizer_steps != (start_epoch - 1) * expected_updates_per_epoch:
            raise RuntimeError('R9 resume optimizer-step count is inconsistent.')
        if best_epoch > 0 and not R9_BEST_CHECKPOINT_PATH.exists():
            raise RuntimeError('R9 best metadata names a missing checkpoint.')
        if (
            diagnostic_best_epoch > 0
            and not R9_DIAGNOSTIC_BEST_CHECKPOINT_PATH.exists()
        ):
            raise RuntimeError(
                'R9 diagnostic-best metadata names a missing checkpoint.'
            )
        resumed = True
    else:
        unexpected = [
            path for path in [
                R9_BEST_CHECKPOINT_PATH,
                R9_DIAGNOSTIC_BEST_CHECKPOINT_PATH,
                R9_HISTORY_PATH,
            ] if path.exists()
        ]
        if unexpected:
            raise RuntimeError(
                'R9 partial artifacts exist without a last checkpoint: '
                + ', '.join(str(path) for path in unexpected)
            )

    if r9_model_digest(model.base) != frozen_base_digest:
        raise RuntimeError('R9 resume changed the frozen R5 base.')
    gc.collect()
    torch.cuda.empty_cache()

    print('\n' + '=' * 80)
    print('TRIAL8 R9 FROZEN-R5 COMPONENT-RESCUE FOLD-0 PILOT STARTING')
    print('=' * 80)
    print(f'Pilot version:                       {R9_PILOT_VERSION}')
    print('Warm start:                          R5 best EMA only')
    print('R7 / R8 weights used:               False / False')
    print('R5 base trainable:                   False (all epochs)')
    print(f'Resume epoch / resumed:              {start_epoch} / {resumed}')
    print('Fit / held-out subjects:             82 / 20')
    print(f'Samples / optimizer updates:         {len(dataset)} / '
          f'{expected_updates_per_epoch} per epoch')
    print(f'Total / trainable parameters:        {expected_parameters:,} / '
          f'{int(preflight_gate["rescue_parameters"]):,}')
    print('Validation and early-stop check:     every epoch')
    print(f'Maximum / minimum epochs / patience: {R9_MAX_EPOCHS} / '
          f'{R9_MINIMUM_EPOCHS} / {R9_EARLY_STOPPING_PATIENCE}')
    print('Final fusion:                        spatial probability union')
    print('Scalar gate:                         disabled')
    print('Fold-0 held-out targets:             validation only')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Full five-fold training authorized:  False')
    print('R5 mean/global/small-case Dice:      '
          f'{baseline["r5_mean_dice"]:.4f} / '
          f'{baseline["r5_global_dice"]:.4f} / '
          f'{baseline["r5_small_case_dice"]:.4f}')
    print('Eligibility floors:                  '
          f'{baseline["mean_dice_floor"]:.4f} / '
          f'{baseline["global_dice_floor"]:.4f} / '
          f'{baseline["small_case_dice_floor"]:.4f}')

    resume_exhausted = bool(
        resumed and start_epoch > R9_MINIMUM_EPOCHS
        and validations_without_improvement >= R9_EARLY_STOPPING_PATIENCE
    )
    stop_reason = 'early_stopping' if resume_exhausted else 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    training_epochs = (
        [] if resume_exhausted
        else range(start_epoch, R9_MAX_EPOCHS + 1)
    )
    for epoch in training_epochs:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True,
        )
        learning_rate = r9_optimizer_lr(epoch)
        for group in optimizer.param_groups:
            group['lr'] = learning_rate
        model.train()
        model.base.eval()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        processed = 0
        epoch_optimizer_steps = 0
        gradient_norms = []
        positive_training_rows = []
        negative_training_rows = []
        dropout_counts = defaultdict(int)
        for batch_index, cpu_batch in enumerate(loader):
            rng = np.random.default_rng(
                R9_SEED + epoch * 1_000_003 + batch_index * 97_409
            )
            dropout_flags = pilot_dropout_flags(rng)
            for name, value in dropout_flags.items():
                dropout_counts[name] += int(bool(value))
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, rows, output = r9_training_loss(
                    model, cpu_batch, device,
                    dropout_flags=dropout_flags,
                )
                scaled_loss = loss / R9_ACCUMULATION_STEPS
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError(f'R9 epoch {epoch}: non-finite loss.')
            scaled_loss.backward()
            if any(parameter.grad is not None for parameter in model.base.parameters()):
                raise RuntimeError(
                    f'R9 epoch {epoch}: gradient reached frozen R5 base.'
                )
            running_loss += float(loss.detach().float().cpu())
            processed += 1
            for row in rows:
                (
                    positive_training_rows
                    if bool(row['positive_case'])
                    else negative_training_rows
                ).append(row)
            del output, loss, scaled_loss
            should_step = (
                (batch_index + 1) % R9_ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.rescue_head.parameters(),
                    R9_GRADIENT_CLIP_NORM,
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'R9 epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                optimizer_steps += 1
                epoch_optimizer_steps += 1
                r9_ema_update(ema_model, model, optimizer_steps)
                gradient_norms.append(float(gradient_norm.cpu()))
        if epoch_optimizer_steps != expected_updates_per_epoch:
            raise RuntimeError(
                f'R9 epoch {epoch}: expected {expected_updates_per_epoch} '
                f'optimizer steps, observed {epoch_optimizer_steps}.'
            )
        if r9_model_digest(model.base) != frozen_base_digest:
            raise RuntimeError(
                f'R9 epoch {epoch}: frozen R5 base state drifted.'
            )
        if r9_model_digest(ema_model.base) != frozen_base_digest:
            raise RuntimeError(
                f'R9 epoch {epoch}: EMA R5 base state drifted.'
            )

        summary, tier_rows = r9_validate(ema_model, epoch, device)
        (
            eligible, guards, selection_key,
            diagnostic_score, metrics,
        ) = r9_checkpoint_evaluation(summary, baseline)
        eligible_improved = bool(
            eligible and (
                best_selection_key is None
                or tuple(selection_key) > tuple(best_selection_key)
            )
        )
        if eligible_improved:
            best_selection_key = list(selection_key)
            best_epoch = int(epoch)
            best_metrics = dict(metrics)
        diagnostic_improved = diagnostic_score > diagnostic_best_score
        if diagnostic_improved:
            diagnostic_best_score = float(diagnostic_score)
            diagnostic_best_epoch = int(epoch)

        patience_improved = (
            diagnostic_score
            > monitor_reference + R9_EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_improved:
            monitor_reference = float(diagnostic_score)
            validations_without_improvement = 0
        elif epoch < R9_MINIMUM_EPOCHS:
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1

        train_center_margin = float(np.mean([
            row['center_margin'] for row in positive_training_rows
            if np.isfinite(float(row['center_margin']))
        ])) if positive_training_rows else float('nan')
        train_extent_margin = float(np.mean([
            row['extent_margin'] for row in positive_training_rows
            if np.isfinite(float(row['extent_margin']))
        ])) if positive_training_rows else float('nan')
        train_rescue_probability = float(np.mean([
            row['exact_rescue_probability_mean']
            for row in positive_training_rows
            if np.isfinite(float(row['exact_rescue_probability_mean']))
        ])) if positive_training_rows else float('nan')
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        history_row = {
            'epoch': int(epoch),
            'train_loss': running_loss / max(processed, 1),
            'train_center_margin': train_center_margin,
            'train_extent_margin': train_extent_margin,
            'train_exact_rescue_probability': train_rescue_probability,
            'train_positive_patches': len(positive_training_rows),
            'train_negative_patches': len(negative_training_rows),
            'learning_rate': float(learning_rate),
            'gradient_norm_mean': float(np.mean(gradient_norms)),
            'optimizer_steps_this_epoch': int(epoch_optimizer_steps),
            'optimizer_steps_total': int(optimizer_steps),
            **{
                f'dropout_count_{name}': int(value)
                for name, value in dropout_counts.items()
            },
            'validation_mean_dice': metrics['positive_mean_dice'],
            'validation_median_dice': float(
                summary['positive_median_dice']
            ),
            'validation_global_dice': metrics['positive_global_dice'],
            'validation_small_case_mean_dice': metrics[
                'small_case_mean_dice'
            ],
            'validation_lesion_precision': float(
                summary['lesion_precision']
            ),
            'validation_lesion_recall': float(summary['lesion_recall']),
            'validation_lesion_f1': metrics['lesion_f1'],
            'validation_component_recall': float(
                summary['component_recall']
            ),
            'validation_small_component_recall': metrics[
                'small_component_recall'
            ],
            'rescue_threshold': float(summary['rescue_threshold']),
            'rescue_component_recall': float(
                summary['rescue_component_recall']
            ),
            'rescue_small_component_recall': float(
                summary['rescue_small_component_recall']
            ),
            'rescue_small_within_2mm': float(
                summary['rescue_small_within_2mm_fraction']
            ),
            'rescue_small_within_4mm': float(
                summary['rescue_small_within_4mm_fraction']
            ),
            'rescue_small_within_8mm': float(
                summary['rescue_small_within_8mm_fraction']
            ),
            'rescue_small_representation_failure': float(
                summary['rescue_small_representation_failure_fraction']
            ),
            'rescue_margin': float(
                summary['rescue_positive_minus_hard_negative_margin']
            ),
            'rescue_voxel_ap': metrics['rescue_ap'],
            'rescue_voxel_precision': float(
                summary['rescue_voxel_precision']
            ),
            'rescue_predicted_to_target_volume_ratio': float(
                summary['rescue_predicted_to_target_volume_ratio']
            ),
            'base_to_final_target_delta': float(
                summary['base_to_final_target_mean_absolute_delta']
            ),
            'base_to_final_background_delta': float(
                summary['base_to_final_background_mean_absolute_delta']
            ),
            **{
                f'validation_{name}_mean_dice': float(
                    bin_metrics.get(name, float('nan'))
                )
                for name in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'local_threshold': float(summary['threshold']),
            'proposal_threshold': float(summary['proposal_threshold']),
            'checkpoint_eligible': bool(eligible),
            **{
                f'eligibility_{name}': bool(value)
                for name, value in guards.items()
            },
            'eligible_best_improved': bool(eligible_improved),
            'best_epoch': int(best_epoch),
            'diagnostic_score': float(diagnostic_score),
            'diagnostic_best_epoch': int(diagnostic_best_epoch),
            'patience_improved': bool(patience_improved),
            'monitor_reference': float(monitor_reference),
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'r5_base_state_exactly_frozen': True,
            'elapsed_minutes': float(elapsed_minutes),
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != int(epoch)
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), R9_HISTORY_PATH)

        checkpoint_record = {
            'pilot_version': R9_PILOT_VERSION,
            'code_revision': R9_CODE_REVISION,
            'model_version': R9_MODEL_VERSION,
            'config_hash': config_hash,
            'preflight_gate_sha256': file_sha256(R9_PREFLIGHT_GATE_PATH),
            'policy_sha256': file_sha256(R9_POLICY_PATH),
            'r5_checkpoint_sha256': r5_checkpoint_sha256,
            'r7_weights_used': False,
            'r8_weights_used': False,
            'r5_frozen': True,
            'parameter_count': expected_parameters,
            'epoch': int(epoch),
            'optimizer_steps': int(optimizer_steps),
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_selection_key': best_selection_key,
            'best_epoch': int(best_epoch),
            'best_metrics': best_metrics,
            'diagnostic_best_epoch': int(diagnostic_best_epoch),
            'diagnostic_best_score': float(diagnostic_best_score),
            'monitor_reference': float(monitor_reference),
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'validation_summary': summary,
            'component_tier_metrics': tier_rows,
            'checkpoint_eligible': bool(eligible),
            'eligibility_guards': guards,
            'selection_key': list(selection_key),
            'diagnostic_score': float(diagnostic_score),
            'train_center_margin': train_center_margin,
            'train_extent_margin': train_extent_margin,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }
        pilot_atomic_torch_save(checkpoint_record, R9_LAST_CHECKPOINT_PATH)
        if eligible_improved:
            pilot_atomic_torch_save(
                checkpoint_record, R9_BEST_CHECKPOINT_PATH
            )
        if diagnostic_improved:
            pilot_atomic_torch_save(
                checkpoint_record, R9_DIAGNOSTIC_BEST_CHECKPOINT_PATH
            )
        atomic_write_json({
            'pilot_version': R9_PILOT_VERSION,
            'code_revision': R9_CODE_REVISION,
            'status': 'running',
            'last_completed_epoch': int(epoch),
            'optimizer_steps': int(optimizer_steps),
            'best_epoch': int(best_epoch),
            'eligible_best_found': best_epoch > 0,
            'diagnostic_best_epoch': int(diagnostic_best_epoch),
            'diagnostic_best_score': float(diagnostic_best_score),
            'validations_without_improvement': int(
                validations_without_improvement
            ),
            'validation_checked_this_epoch': True,
            'r5_base_state_exactly_frozen': True,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'full_fivefold_training_authorized': False,
        }, R9_STATE_PATH)

        print(
            f'Epoch {epoch:03d}/{R9_MAX_EPOCHS}: '
            f'loss={running_loss/max(processed,1):.4f}, '
            f'train center/extent margin={train_center_margin:.4f}/'
            f'{train_extent_margin:.4f}, '
            f'lr={learning_rate:.2e}, '
            f'grad={np.mean(gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            '  Validation: mean/median/global Dice='
            f'{summary["positive_mean_dice"]:.4f}/'
            f'{summary["positive_median_dice"]:.4f}/'
            f'{summary["positive_global_dice"]:.4f}, '
            'lesion P/R/F1='
            f'{summary["lesion_precision"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}/'
            f'{summary["lesion_f1"]:.4f}'
        )
        print(
            '  Final small cases/components: Dice='
            f'{summary["small_case_mean_dice"]:.4f}, recall='
            f'{summary["small_component_recall"]:.4f}; '
            f'all-component recall={summary["component_recall"]:.4f}'
        )
        print(
            '  Rescue detector: threshold/AP/precision/ratio='
            f'{summary["rescue_threshold"]:.2f}/'
            f'{summary["rescue_voxel_average_precision_256bin"]:.4f}/'
            f'{summary["rescue_voxel_precision"]:.4f}/'
            f'{summary["rescue_predicted_to_target_volume_ratio"]:.3f}, '
            'small recall/within 2/4/8 mm='
            f'{summary["rescue_small_component_recall"]:.4f}/'
            f'{summary["rescue_small_within_2mm_fraction"]:.4f}/'
            f'{summary["rescue_small_within_4mm_fraction"]:.4f}/'
            f'{summary["rescue_small_within_8mm_fraction"]:.4f}'
        )
        print(
            '  Transfer: held-out margin/representation failure/'
            'target delta/background delta='
            f'{summary["rescue_positive_minus_hard_negative_margin"]:.4f}/'
            f'{summary["rescue_small_representation_failure_fraction"]:.4f}/'
            f'{summary["base_to_final_target_mean_absolute_delta"]:.5f}/'
            f'{summary["base_to_final_background_mean_absolute_delta"]:.5f}'
        )
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}; '
            f'eligible={eligible}, guards={guards}, '
            f'best={best_epoch}, diagnostic_best={diagnostic_best_epoch}, '
            f'patience={validations_without_improvement}/'
            f'{R9_EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if (
            epoch >= R9_MINIMUM_EPOCHS
            and validations_without_improvement
            >= R9_EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if last_summary is None:
        if not R9_LAST_CHECKPOINT_PATH.exists():
            raise RuntimeError('R9 training produced no checkpoint.')
        checkpoint = torch.load(
            R9_LAST_CHECKPOINT_PATH, map_location='cpu',
            weights_only=False,
        )
        last_summary = dict(checkpoint['validation_summary'])
        last_tier_rows = list(checkpoint['component_tier_metrics'])

    completed_epoch = max(int(row['epoch']) for row in history_rows)
    eligible_best_found = bool(
        best_epoch > 0 and R9_BEST_CHECKPOINT_PATH.exists()
    )
    selected_checkpoint_path = (
        R9_BEST_CHECKPOINT_PATH
        if eligible_best_found else R9_DIAGNOSTIC_BEST_CHECKPOINT_PATH
    )
    if not selected_checkpoint_path.exists():
        raise RuntimeError('R9 selected checkpoint is missing.')
    selected_checkpoint = torch.load(
        selected_checkpoint_path, map_location='cpu', weights_only=False
    )
    selected_summary = dict(selected_checkpoint['validation_summary'])
    selected_tier_rows = list(
        selected_checkpoint['component_tier_metrics']
    )
    selected_epoch = int(selected_checkpoint['epoch'])
    selected_metrics = r9_checkpoint_evaluation(
        selected_summary, baseline
    )
    selected_eligible = bool(selected_metrics[0])
    if selected_eligible != eligible_best_found:
        raise RuntimeError('R9 final checkpoint eligibility disagrees.')

    directional_checks = {
        'eligible_checkpoint_found': eligible_best_found,
        'mean_dice_at_least_r5': (
            float(selected_summary['positive_mean_dice'])
            >= baseline['r5_mean_dice']
        ),
        'small_component_recall_improved': (
            float(selected_summary['small_component_recall'])
            > baseline['r5_small_component_recall']
        ),
        'small_case_dice_not_below_floor': (
            float(selected_summary['small_case_mean_dice'])
            >= baseline['small_case_dice_floor']
        ),
        'heldout_rescue_margin_positive': (
            float(selected_summary[
                'rescue_positive_minus_hard_negative_margin'
            ]) > 0.0
        ),
        'rescue_changes_targets_more_than_background': (
            float(selected_summary[
                'base_to_final_target_mean_absolute_delta'
            ]) > float(selected_summary[
                'base_to_final_background_mean_absolute_delta'
            ])
        ),
        'r5_base_remained_exactly_frozen': (
            r9_model_digest(model.base) == frozen_base_digest
            and r9_model_digest(ema_model.base) == frozen_base_digest
        ),
    }
    directional_success = all(directional_checks.values())
    if directional_success:
        recommendation = (
            'R9 succeeded directionally on fold 0. Build a separate '
            'fold-replication preflight; keep locked validation and test sealed.'
        )
    elif eligible_best_found:
        recommendation = (
            'R9 produced a guarded candidate but not a clean directional '
            'success. Retain R5 for deployment and report R9 as an ablation.'
        )
    else:
        recommendation = (
            'Retain R5. The component detector did not improve held-out small '
            'component recall within the preserved-metric guards; stop '
            'architecture iteration and proceed to results reporting.'
        )

    result = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_complete': True,
        'pilot_version': R9_PILOT_VERSION,
        'code_revision': R9_CODE_REVISION,
        'model_version': R9_MODEL_VERSION,
        'stop_reason': stop_reason,
        'completed_epoch': int(completed_epoch),
        'selected_epoch': int(selected_epoch),
        'best_epoch': int(best_epoch) if eligible_best_found else None,
        'diagnostic_best_epoch': int(diagnostic_best_epoch),
        'eligible_r9_checkpoint_found': eligible_best_found,
        'directional_success': bool(directional_success),
        'directional_checks': directional_checks,
        'selected_validation_summary': selected_summary,
        'selected_component_tier_metrics': selected_tier_rows,
        'last_validation_summary': last_summary,
        'last_component_tier_metrics': last_tier_rows,
        'r5_baseline': baseline,
        'mean_dice_change_vs_r5': (
            float(selected_summary['positive_mean_dice'])
            - baseline['r5_mean_dice']
        ),
        'global_dice_change_vs_r5': (
            float(selected_summary['positive_global_dice'])
            - baseline['r5_global_dice']
        ),
        'small_case_dice_change_vs_r5': (
            float(selected_summary['small_case_mean_dice'])
            - baseline['r5_small_case_dice']
        ),
        'small_component_recall_change_vs_r5': (
            float(selected_summary['small_component_recall'])
            - baseline['r5_small_component_recall']
        ),
        'recommendation': recommendation,
        'preflight_gate_sha256': file_sha256(R9_PREFLIGHT_GATE_PATH),
        'policy_sha256': file_sha256(R9_POLICY_PATH),
        'config_sha256': file_sha256(R9_CONFIG_PATH),
        'r5_checkpoint_sha256': r5_checkpoint_sha256,
        'last_checkpoint_sha256': file_sha256(R9_LAST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': (
            file_sha256(R9_BEST_CHECKPOINT_PATH)
            if eligible_best_found else None
        ),
        'diagnostic_best_checkpoint_sha256': file_sha256(
            R9_DIAGNOSTIC_BEST_CHECKPOINT_PATH
        ),
        'deployment_checkpoint': (
            str(R9_BEST_CHECKPOINT_PATH)
            if directional_success else str(R5_BEST_CHECKPOINT_PATH)
        ),
        'fold0_heldout_targets_read_for_validation': True,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'full_fivefold_training_authorized': False,
        'fold_replication_preflight_allowed': bool(directional_success),
    }
    atomic_write_json(result, R9_FINAL_SUMMARY_PATH)
    result['final_summary_sha256'] = file_sha256(R9_FINAL_SUMMARY_PATH)
    atomic_write_json(result, R9_RESULT_GATE_PATH)

    print('\n' + '=' * 80)
    print('TRIAL8 R9 FROZEN-R5 COMPONENT-RESCUE FOLD-0 PILOT COMPLETE')
    print('=' * 80)
    print(f'Stop reason / last epoch:           {stop_reason} / {completed_epoch}')
    print(f'Eligible R9 checkpoint found:       {eligible_best_found}')
    print(f'Selected epoch:                     {selected_epoch}')
    print('Selected mean/global Dice:          '
          f'{float(selected_summary["positive_mean_dice"]):.4f} / '
          f'{float(selected_summary["positive_global_dice"]):.4f}')
    print('Selected small Dice/component recall:'
          f' {float(selected_summary["small_case_mean_dice"]):.4f} / '
          f'{float(selected_summary["small_component_recall"]):.4f}')
    print('R5 mean/global/small Dice:           '
          f'{baseline["r5_mean_dice"]:.4f} / '
          f'{baseline["r5_global_dice"]:.4f} / '
          f'{baseline["r5_small_case_dice"]:.4f}')
    print('Rescue margin / AP:                 '
          f'{float(selected_summary["rescue_positive_minus_hard_negative_margin"]):.4f} / '
          f'{float(selected_summary["rescue_voxel_average_precision_256bin"]):.4f}')
    print(f'Directional success:                {directional_success}')
    print(f'Recommended next action:            {recommendation}')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Full five-fold training authorized: False')
    print('Deployment checkpoint:              '
          f'{result["deployment_checkpoint"]}')
    print(f'Last checkpoint:                    {R9_LAST_CHECKPOINT_PATH}')
    print(f'History:                            {R9_HISTORY_PATH}')
    print(f'Result gate:                        {R9_RESULT_GATE_PATH}')


run_r9_preflight()
run_r9_fold0_pilot()


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/mnt/storage')
except ImportError:
    pass

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, 'to_string') else value)






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
R5_MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
R5_PILOT_VERSION = 'v2_2_fold0_global_local_component_diagnostic_r5'
R5_CODE_REVISION = 'r5_fresh_start_dice_iou_selection'

RECONCILIATION_VERSION = 'v1_r5_metric_reconciliation_fivefold_authorization'
CODE_REVISION = 'r2_cross_contract_semantics_completed_r8'
METRIC_CONTRACT_REVISION = 'trial8_final_reporting_metrics_v1'







CANONICAL_RECONCILIATION_TOLERANCE = 1e-6
CROSS_CONTRACT_RECOMPUTATION_TOLERANCE = 1e-4

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_R5_PARAMETERS = 16_870_679
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)

FIVEFOLD_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
    / 'fivefold_loader_loss_sampler_gate.json'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V22_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / R5_MODEL_VERSION / 'v2_2_global_local_model_construction_r2'
    / 'v2_2_model_construction_gate.json'
)

R5_ROOT = PROJECT_ROOT / 'checkpoints' / R5_MODEL_VERSION / R5_PILOT_VERSION
R5_RESULT_GATE_PATH = R5_ROOT / 'pilot_result_gate.json'
R5_BEST_CHECKPOINT_PATH = R5_ROOT / 'best_checkpoint.pt'
R5_HISTORY_PATH = R5_ROOT / 'training_history.csv'

R7_ROOT = (
    PROJECT_ROOT / 'checkpoints' / R5_MODEL_VERSION
    / 'v2_2_fold0_local_contrastive_proposal_r7'
)
R7_RESULT_GATE_PATH = R7_ROOT / 'pilot_result_gate.json'
R7_R5_BASELINE_DIAGNOSTIC_PATH = (
    R7_ROOT / 'diagnostics' / 'r5_volume_bounded_baseline.json'
)

R8_ROOT = (
    PROJECT_ROOT / 'checkpoints'
    / 'trial8_vascufate_net_v2_3_dual_resolution_satellite'
    / 'v2_3_fold0_dual_resolution_satellite_r8'
)
R8_RESULT_GATE_PATH = R8_ROOT / 'pilot_result_gate.json'
R8_HISTORY_PATH = R8_ROOT / 'training_history.csv'
R8_FINAL_SUMMARY_PATH = R8_ROOT / 'final_pilot_summary.json'

R9_ROOT = (
    PROJECT_ROOT / 'checkpoints'
    / 'trial8_vascufate_net_v2_4_component_rescue'
    / 'v2_4_fold0_component_center_extent_rescue_r9'
)
R9_RESULT_GATE_PATH = R9_ROOT / 'pilot_result_gate.json'
R9_FINAL_SUMMARY_PATH = R9_ROOT / 'final_pilot_summary.json'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / R5_MODEL_VERSION / RECONCILIATION_VERSION
)
METRIC_DEFINITIONS_PATH = OUTPUT_ROOT / 'metric_definitions.json'
METRIC_RECONCILIATION_PATH = OUTPUT_ROOT / 'metric_reconciliation.csv'
ARCHITECTURE_SELECTION_PATH = OUTPUT_ROOT / 'architecture_selection.csv'
CROSS_CONTRACT_AUDIT_PATH = OUTPUT_ROOT / 'cross_contract_metric_audit.csv'
AUTHORIZATION_POLICY_PATH = OUTPUT_ROOT / 'fivefold_evaluation_policy.json'
CHECKS_PATH = OUTPUT_ROOT / 'reconciliation_checks.csv'
AUTHORIZATION_GATE_PATH = (
    OUTPUT_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
OUTPUT_ARTIFACTS = [
    METRIC_DEFINITIONS_PATH,
    METRIC_RECONCILIATION_PATH,
    ARCHITECTURE_SELECTION_PATH,
    CROSS_CONTRACT_AUDIT_PATH,
    AUTHORIZATION_POLICY_PATH,
    CHECKS_PATH,
]






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_builtin(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    if isinstance(value, (int, np.integer, float, np.floating)):
        return bool(value)
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot interpret as boolean: {value!r}')


def close(left, right, tolerance=1e-6):
    return bool(math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=tolerance
    ))


def metric_value(record, *keys, default=float('nan')):
    current = record
    for key in keys:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    try:
        return float(current)
    except (TypeError, ValueError):
        return default


def completed_gate_is_stable():
    if not AUTHORIZATION_GATE_PATH.exists():
        return False
    gate = load_json(AUTHORIZATION_GATE_PATH)
    if not (
        gate.get('authorization_passed') is True
        and gate.get('reconciliation_version') == RECONCILIATION_VERSION
        and gate.get('code_revision') == CODE_REVISION
        and gate.get('metric_contract_revision') == METRIC_CONTRACT_REVISION
    ):
        return False
    output_hashes = gate.get('output_sha256', {})
    outputs_stable = all(
        path.exists()
        and output_hashes.get(path.name) == file_sha256(path)
        for path in OUTPUT_ARTIFACTS
    )
    input_hashes = gate.get('input_sha256', {})
    inputs_stable = bool(input_hashes) and all(
        Path(path).exists() and file_sha256(Path(path)) == digest
        for path, digest in input_hashes.items()
    )
    return outputs_stable and inputs_stable






def run_reconciliation():
    if completed_gate_is_stable():
        gate = load_json(AUTHORIZATION_GATE_PATH)
        print('\n' + '=' * 80)
        print('TRIAL8 R5 METRIC RECONCILIATION ALREADY COMPLETE')
        print('=' * 80)
        print(f'Selected checkpoint: {gate["selected_checkpoint_path"]}')
        print(f'Authorization gate:  {AUTHORIZATION_GATE_PATH}')
        print('No source artifact or authorization output was rewritten.')
        return

    if AUTHORIZATION_GATE_PATH.exists():
        existing = load_json(AUTHORIZATION_GATE_PATH)
        replaceable_failed_lineage = (
            existing.get('authorization_passed') is False
            and existing.get('reconciliation_version')
            == RECONCILIATION_VERSION
        )
        if not replaceable_failed_lineage:
            raise RuntimeError(
                'An incompatible reconciliation gate already exists. Refusing '
                'to overwrite a completed selection decision.'
            )
        print(
            'Replacing a failed authorization gate from revision '
            f'{existing.get("code_revision", "unknown")} with {CODE_REVISION}; '
            'no model or checkpoint artifact will be changed.'
        )

    required_paths = [
        TARGET_POLICY_MANIFEST_PATH,
        FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH,
        GRAPH_SCHEMA_PATH,
        FIVEFOLD_GATE_PATH,
        MULTIMODAL_GATE_PATH,
        V22_MODEL_GATE_PATH,
        R5_RESULT_GATE_PATH,
        R5_BEST_CHECKPOINT_PATH,
        R5_HISTORY_PATH,
        R7_RESULT_GATE_PATH,
        R7_R5_BASELINE_DIAGNOSTIC_PATH,
        R8_RESULT_GATE_PATH,
        R8_HISTORY_PATH,
        R8_FINAL_SUMMARY_PATH,
        R9_RESULT_GATE_PATH,
        R9_FINAL_SUMMARY_PATH,
    ]
    missing = [str(path) for path in required_paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'Required reconciliation inputs are missing:\n' + '\n'.join(missing)
        )

    target_policy_hash = file_sha256(TARGET_POLICY_MANIFEST_PATH)
    fold_manifest_hash = file_sha256(FOLD_MANIFEST_PATH)
    multimodal_manifest_hash = file_sha256(MULTIMODAL_MANIFEST_PATH)
    graph_schema_hash = file_sha256(GRAPH_SCHEMA_PATH)
    r5_checkpoint_hash_before = file_sha256(R5_BEST_CHECKPOINT_PATH)

    fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    model_gate = load_json(V22_MODEL_GATE_PATH)
    r5_gate = load_json(R5_RESULT_GATE_PATH)
    r7_gate = load_json(R7_RESULT_GATE_PATH)
    r7_r5_baseline = load_json(R7_R5_BASELINE_DIAGNOSTIC_PATH)
    r8_result = load_json(R8_RESULT_GATE_PATH)
    r9_gate = load_json(R9_RESULT_GATE_PATH)
    r9_summary = load_json(R9_FINAL_SUMMARY_PATH)

    best_epoch = int(r5_gate.get('best_epoch', -1))
    if best_epoch <= 0:
        raise RuntimeError('R5 result gate does not contain a valid best epoch.')
    r5_epoch_root = R5_ROOT / 'validation' / f'epoch_{best_epoch:03d}'
    r5_case_metrics_path = r5_epoch_root / 'case_metrics.csv'
    r5_tier_metrics_path = r5_epoch_root / 'component_tier_metrics.csv'
    r5_validation_summary_path = r5_epoch_root / 'validation_summary.json'
    dynamic_required = [
        r5_case_metrics_path,
        r5_tier_metrics_path,
        r5_validation_summary_path,
    ]
    dynamic_missing = [str(path) for path in dynamic_required if not path.exists()]
    if dynamic_missing:
        raise FileNotFoundError(
            'R5 best-epoch validation artifacts are missing:\n'
            + '\n'.join(dynamic_missing)
        )

    r5_summary = load_json(r5_validation_summary_path)
    r5_gate_summary = dict(r5_gate.get('best_validation_summary', {}))
    r7_summary = dict(r7_r5_baseline.get('validation_summary', {}))
    r9_selected = dict(r9_gate.get('selected_validation_summary', {}))
    r9_baseline = dict(r9_gate.get('r5_baseline', {}))

    target_policy_df = pd.read_csv(TARGET_POLICY_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    multimodal_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    case_df = pd.read_csv(r5_case_metrics_path)
    tier_df = pd.read_csv(r5_tier_metrics_path)
    history_df = pd.read_csv(R5_HISTORY_PATH)

    for frame, name in [
        (target_policy_df, 'target-policy manifest'),
        (fold_df, 'fold manifest'),
        (multimodal_df, 'multimodal manifest'),
        (case_df, 'R5 case metrics'),
    ]:
        if 'subject_id' not in frame.columns:
            raise RuntimeError(f'{name} lacks subject_id.')
        frame['subject_id'] = frame['subject_id'].astype(str)

    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold'
    }
    required_case_columns = {
        'subject_id', 'lesion_volume_bin', 'hard_metric_eligible',
        'target_voxels', 'predicted_voxels', 'dice'
    }
    required_tier_columns = {
        'component_tier', 'components', 'final_component_recall'
    }
    if not required_fold_columns.issubset(fold_df.columns):
        raise RuntimeError('Fold manifest schema is incomplete.')
    if not required_case_columns.issubset(case_df.columns):
        raise RuntimeError('R5 case-metric schema is incomplete.')
    if not required_tier_columns.issubset(tier_df.columns):
        raise RuntimeError('R5 component-tier schema is incomplete.')

    split_counts = (
        multimodal_df['dev_split'].astype(str).value_counts().to_dict()
    )
    target_policy_subjects = set(target_policy_df['subject_id'].astype(str))
    fold_manifest_subjects = set(fold_df['subject_id'].astype(str))
    multimodal_subjects = set(multimodal_df['subject_id'].astype(str))
    development_folds = fold_df[
        fold_df['frozen_dev_split'].astype(str) == 'train'
    ].copy()
    fold_counts = (
        development_folds['development_fold'].astype(int)
        .value_counts().sort_index().to_dict()
    )
    fold0_subjects = set(
        development_folds.loc[
            development_folds['development_fold'].astype(int) == 0,
            'subject_id',
        ].astype(str)
    )
    locked_validation_subjects = set(
        multimodal_df.loc[
            multimodal_df['dev_split'].astype(str) == 'val', 'subject_id'
        ].astype(str)
    )
    sealed_test_subjects = set(
        multimodal_df.loc[
            multimodal_df['dev_split'].astype(str) == 'test', 'subject_id'
        ].astype(str)
    )
    metric_subjects = set(case_df['subject_id'].astype(str))

    case_df['hard_metric_eligible_bool'] = case_df[
        'hard_metric_eligible'
    ].map(as_bool)
    case_df['target_voxels'] = pd.to_numeric(
        case_df['target_voxels'], errors='raise'
    ).astype(int)
    case_df['predicted_voxels'] = pd.to_numeric(
        case_df['predicted_voxels'], errors='raise'
    ).astype(int)
    case_df['dice'] = pd.to_numeric(case_df['dice'], errors='raise')

    positive = case_df[
        case_df['hard_metric_eligible_bool']
        & (case_df['target_voxels'] > 0)
    ].copy()
    small_cases = positive[
        positive['lesion_volume_bin'].astype(str).isin({'tiny', 'small'})
    ].copy()
    tiny_cases = positive[
        positive['lesion_volume_bin'].astype(str) == 'tiny'
    ].copy()
    small_bin_cases = positive[
        positive['lesion_volume_bin'].astype(str) == 'small'
    ].copy()
    if len(tiny_cases) == 0 or len(small_bin_cases) == 0:
        raise RuntimeError(
            'Fold-0 held-out metrics must contain both tiny and small cases.'
        )

    positive_mean = float(positive['dice'].mean())
    positive_median = float(positive['dice'].median())
    tiny_mean = float(tiny_cases['dice'].mean())
    small_bin_mean = float(small_bin_cases['dice'].mean())
    small_subject_weighted = float(small_cases['dice'].mean())
    small_bin_macro = float(np.mean([tiny_mean, small_bin_mean]))

    intersection_float = (
        positive['dice'].to_numpy(dtype=np.float64)
        * (
            positive['predicted_voxels'].to_numpy(dtype=np.float64)
            + positive['target_voxels'].to_numpy(dtype=np.float64)
        )
        / 2.0
    )
    intersection_round = np.rint(intersection_float)
    intersections_reconstruct_to_integer = bool(np.allclose(
        intersection_float, intersection_round, rtol=0.0, atol=1e-5
    ))
    positive_global = float(
        2.0 * intersection_round.sum()
        / max(
            int(positive['predicted_voxels'].sum())
            + int(positive['target_voxels'].sum()),
            1,
        )
    )

    small_tiers = {
        'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
    }
    selected_tiers = tier_df[
        tier_df['component_tier'].astype(str).isin(small_tiers)
    ].copy()
    selected_tiers['components'] = pd.to_numeric(
        selected_tiers['components'], errors='raise'
    ).astype(int)
    selected_tiers['final_component_recall'] = pd.to_numeric(
        selected_tiers['final_component_recall'], errors='raise'
    )
    small_component_denominator = int(selected_tiers['components'].sum())
    small_component_numerator = float(
        (
            selected_tiers['components']
            * selected_tiers['final_component_recall']
        ).sum()
    )
    small_component_recall = float(
        small_component_numerator / max(small_component_denominator, 1)
    )

    r9_reported_bin_macro = metric_value(
        r9_gate, 'r5_baseline', 'r5_small_case_dice'
    )
    r5_reported_subject_weighted_small = metric_value(
        r5_summary, 'small_case_mean_dice'
    )

    metric_definitions = {
        'metric_contract_revision': METRIC_CONTRACT_REVISION,
        'primary_metrics': {
            'positive_case_mean_dice': (
                'Arithmetic mean of subject-level Dice over hard-metric-eligible '
                'subjects with nonempty targets.'
            ),
            'positive_case_median_dice': (
                'Median subject-level Dice over the same positive-subject set.'
            ),
            'pooled_global_dice': (
                'Dice after pooling intersections, predictions, and targets '
                'across hard-metric-eligible positive subjects.'
            ),
            'tiny_small_subject_weighted_mean_dice': (
                'Arithmetic mean of subject-level Dice across every eligible '
                'tiny or small case. Each subject receives equal weight. This '
                'is the canonical small-case reporting and selection metric.'
            ),
            'small_component_recall': (
                'Component-count-weighted recall across subvoxel/single-voxel, '
                'micro, tiny, and small target components using final-mask '
                'IoU >= 0.10.'
            ),
        },
        'secondary_sensitivity_metrics': {
            'tiny_small_bin_macro_mean_dice': (
                'Unweighted mean of the tiny-bin mean Dice and small-bin mean '
                'Dice. Each bin receives equal weight regardless of subject '
                'count. Historical value approximately 0.0347.'
            ),
        },
        'prohibited_ambiguous_label': 'small Dice',
        'historical_discrepancy_resolution': {
            'approximately_0p0579': (
                'tiny_small_subject_weighted_mean_dice'
            ),
            'approximately_0p0347': 'tiny_small_bin_macro_mean_dice',
            'values_are_not_contradictory': True,
        },
        'selection_policy': (
            'Use subject-weighted subgroup means as primary metrics so subgroup '
            'aggregation matches the overall mean-Dice definition. Report the '
            'equal-bin macro mean as a named secondary sensitivity analysis.'
        ),
    }

    reconciliation_rows = [
        {
            'metric': 'positive_case_mean_dice',
            'recomputed_value': positive_mean,
            'stored_value': metric_value(r5_summary, 'positive_mean_dice'),
            'aggregation': 'subject_weighted',
            'role': 'primary',
        },
        {
            'metric': 'positive_case_median_dice',
            'recomputed_value': positive_median,
            'stored_value': metric_value(r5_summary, 'positive_median_dice'),
            'aggregation': 'subject_median',
            'role': 'primary',
        },
        {
            'metric': 'pooled_global_dice',
            'recomputed_value': positive_global,
            'stored_value': metric_value(r5_summary, 'positive_global_dice'),
            'aggregation': 'pooled_voxels',
            'role': 'primary',
        },
        {
            'metric': 'tiny_bin_mean_dice',
            'recomputed_value': tiny_mean,
            'stored_value': metric_value(
                r5_summary, 'lesion_bin_mean_dice', 'tiny'
            ),
            'aggregation': 'within_bin_subject_weighted',
            'role': 'stratified',
        },
        {
            'metric': 'small_bin_mean_dice',
            'recomputed_value': small_bin_mean,
            'stored_value': metric_value(
                r5_summary, 'lesion_bin_mean_dice', 'small'
            ),
            'aggregation': 'within_bin_subject_weighted',
            'role': 'stratified',
        },
        {
            'metric': 'tiny_small_subject_weighted_mean_dice',
            'recomputed_value': small_subject_weighted,
            'stored_value': metric_value(r5_summary, 'small_case_mean_dice'),
            'aggregation': 'all_tiny_small_subjects_equal_weight',
            'role': 'primary',
        },
        {
            'metric': 'tiny_small_bin_macro_mean_dice',
            'recomputed_value': small_bin_macro,
            'stored_value': r9_reported_bin_macro,
            'aggregation': 'tiny_and_small_bins_equal_weight',
            'role': 'secondary_sensitivity',
        },
        {
            'metric': 'small_component_recall',
            'recomputed_value': small_component_recall,
            'stored_value': metric_value(r5_summary, 'small_component_recall'),
            'aggregation': 'component_count_weighted',
            'role': 'primary',
        },
    ]
    reconciliation_df = pd.DataFrame(reconciliation_rows)
    reconciliation_df['absolute_error'] = np.abs(
        reconciliation_df['recomputed_value']
        - reconciliation_df['stored_value']
    )
    reconciliation_df['matches'] = (
        reconciliation_df['absolute_error']
        <= CANONICAL_RECONCILIATION_TOLERANCE
    )

    cross_contract_rows = []
    cross_contract_specs = [
        (
            'r7_readonly_r5_positive_mean_dice',
            metric_value(r7_summary, 'positive_mean_dice'),
            positive_mean,
            'R7 read-only R5 checkpoint re-inference',
            True,
        ),
        (
            'r7_readonly_r5_pooled_global_dice',
            metric_value(r7_summary, 'positive_global_dice'),
            positive_global,
            'R7 read-only R5 checkpoint re-inference',
            True,
        ),
        (
            'r7_readonly_r5_tiny_small_subject_mean_dice',
            metric_value(r7_summary, 'small_case_mean_dice'),
            small_subject_weighted,
            'R7 read-only R5 checkpoint re-inference',
            True,
        ),
        (
            'r9_baseline_positive_mean_dice',
            metric_value(r9_baseline, 'r5_mean_dice'),
            metric_value(r7_summary, 'positive_mean_dice'),
            'R9 frozen baseline copied from the R7 R5 recomputation',
            True,
        ),
        (
            'r9_baseline_pooled_global_dice',
            metric_value(r9_baseline, 'r5_global_dice'),
            metric_value(r7_summary, 'positive_global_dice'),
            'R9 frozen baseline copied from the R7 R5 recomputation',
            True,
        ),
        (
            'r9_baseline_tiny_small_bin_macro_dice',
            metric_value(r9_baseline, 'r5_small_case_dice'),
            small_bin_macro,
            'Historical R9 baseline used equal tiny/small bin weighting',
            True,
        ),
        (
            'r9_rejected_candidate_positive_mean_dice',
            metric_value(r9_selected, 'positive_mean_dice'),
            positive_mean,
            'Rejected R9 diagnostic candidate; not the deployment output',
            False,
        ),
        (
            'r9_rejected_candidate_pooled_global_dice',
            metric_value(r9_selected, 'positive_global_dice'),
            positive_global,
            'Rejected R9 diagnostic candidate; not the deployment output',
            False,
        ),
        (
            'r9_rejected_candidate_tiny_small_subject_mean_dice',
            metric_value(r9_selected, 'small_case_mean_dice'),
            small_subject_weighted,
            'Rejected R9 diagnostic candidate; not the deployment output',
            False,
        ),
    ]
    for (
        name, observed, reference, provenance, required_for_authorization
    ) in cross_contract_specs:
        error = abs(float(observed) - float(reference))
        cross_contract_rows.append({
            'comparison': name,
            'observed_value': float(observed),
            'reference_value': float(reference),
            'absolute_error': error,
            'tolerance': CROSS_CONTRACT_RECOMPUTATION_TOLERANCE,
            'matches': bool(
                np.isfinite(observed)
                and np.isfinite(reference)
                and error <= CROSS_CONTRACT_RECOMPUTATION_TOLERANCE
            ),
            'required_for_authorization': bool(required_for_authorization),
            'provenance': provenance,
        })
    cross_contract_df = pd.DataFrame(cross_contract_rows)

    r8_history = pd.read_csv(R8_HISTORY_PATH)
    r8_history_rows = int(len(r8_history))
    r8_last_epoch = 0
    if len(r8_history) and 'epoch' in r8_history.columns:
        r8_last_epoch = int(pd.to_numeric(
            r8_history['epoch'], errors='raise'
        ).max())
    r8_complete = r8_result.get('pilot_complete') is True
    r8_directional_success = r8_result.get('directional_success') is True
    r8_replication_allowed = (
        r8_result.get('fold_replication_preflight_allowed') is True
    )
    r8_accepted = bool(
        r8_complete and r8_directional_success and r8_replication_allowed
    )
    r8_status = (
        'completed_rejected' if r8_complete and not r8_accepted
        else 'completed_candidate' if r8_complete
        else 'incomplete_excluded'
    )
    if r8_complete and not r8_accepted:
        r8_reason = (
            'Completed but failed directional/replication gates; '
            'eligible checkpoint='
            f'{bool(r8_result.get("eligible_r8_checkpoint_found"))}, '
            f'stop={r8_result.get("stop_reason", "unknown")}, '
            'last epoch='
            f'{r8_result.get("last_completed_epoch", r8_last_epoch)}.'
        )
    elif r8_complete:
        r8_reason = (
            'Completed with directional and replication gates true; this '
            'must block automatic R5 authorization for manual review.'
        )
    else:
        r8_reason = (
            f'No completed result; observed {r8_history_rows} history rows '
            f'through epoch {r8_last_epoch}.'
        )

    architecture_df = pd.DataFrame([
        {
            'experiment': 'R5',
            'status': 'selected',
            'complete': bool(r5_gate.get('pilot_complete') is True),
            'eligible_for_selection': True,
            'weight_source': 'R5 best EMA',
            'reason': 'Best completed guarded development architecture.',
        },
        {
            'experiment': 'R7',
            'status': 'rejected_ablation',
            'complete': bool(r7_gate.get('pilot_complete') is True),
            'eligible_for_selection': False,
            'weight_source': 'R7',
            'reason': 'Directional success gates failed.',
        },
        {
            'experiment': 'R8',
            'status': r8_status,
            'complete': bool(r8_complete),
            'eligible_for_selection': False,
            'weight_source': 'R8',
            'reason': r8_reason,
        },
        {
            'experiment': 'R9',
            'status': 'rejected_ablation',
            'complete': bool(r9_gate.get('pilot_complete') is True),
            'eligible_for_selection': False,
            'weight_source': 'R9 rescue head over frozen R5',
            'reason': (
                'No eligible checkpoint and no held-out small-component recall '
                'improvement.'
            ),
        },
    ])

    authorization_policy = {
        'policy_version': RECONCILIATION_VERSION,
        'metric_contract_revision': METRIC_CONTRACT_REVISION,
        'selected_architecture': 'R5 VascuFateNet V2.2 global-local arterial',
        'selected_model_version': R5_MODEL_VERSION,
        'selected_pilot_version': R5_PILOT_VERSION,
        'selected_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'selected_checkpoint_sha256': r5_checkpoint_hash_before,
        'architecture_parameters': EXPECTED_R5_PARAMETERS,
        'canonical_metric_source': str(r5_validation_summary_path),
        'canonical_case_metric_source': str(r5_case_metrics_path),
        'canonical_reconciliation_tolerance': (
            CANONICAL_RECONCILIATION_TOLERANCE
        ),
        'independent_recomputation_tolerance': (
            CROSS_CONTRACT_RECOMPUTATION_TOLERANCE
        ),
        'later_rejected_candidate_summaries_are_not_deployment_metrics': True,
        'architecture_frozen': True,
        'hyperparameters_frozen_from_r5': True,
        'loss_and_sampling_contract_frozen_from_r5': True,
        'new_training_authorized_folds': [1, 2, 3, 4],
        'fold0_artifacts_reused': True,
        'fold0_is_architecture_development_fold': True,
        'development_subjects': 102,
        'fold_specific_scalers_required': True,
        'scalers_fit_on_each_fold_fit_partition_only': True,
        'fold_heldout_targets_allowed_for_epoch_selection': True,
        'locked_validation_targets_authorized': False,
        'test_targets_authorized': False,
        'out_of_fold_threshold_policy': (
            'After all five development folds, choose one frozen threshold '
            'from pooled out-of-fold predictions; do not tune on locked '
            'validation or test targets.'
        ),
        'reporting_caveat': (
            'Fold 0 was used for architecture development. Development '
            'fivefold summaries characterize stability but are not the final '
            'independent performance estimate. Locked validation remains the '
            'first independent confirmation set.'
        ),
        'architecture_iteration_closed': True,
        'r7_r8_r9_weights_prohibited': True,
    }

    fold0_case_subjects_match = metric_subjects == fold0_subjects
    no_locked_subjects_in_metrics = not bool(
        metric_subjects & locked_validation_subjects
    )
    no_test_subjects_in_metrics = not bool(metric_subjects & sealed_test_subjects)
    r5_summary_required = {
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'small_case_mean_dice',
        'small_component_recall', 'lesion_bin_mean_dice'
    }
    r5_history_best_present = bool(
        'epoch' in history_df.columns
        and best_epoch in set(pd.to_numeric(
            history_df['epoch'], errors='raise'
        ).astype(int))
    )

    checks = {
        'target_policy_manifest_hash_matches': (
            target_policy_hash == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
        ),
        'fold_manifest_hash_matches': (
            fold_manifest_hash == EXPECTED_FOLD_MANIFEST_SHA256
        ),
        'multimodal_manifest_hash_matches': (
            multimodal_manifest_hash == EXPECTED_MULTIMODAL_MANIFEST_SHA256
        ),
        'graph_schema_hash_matches': (
            graph_schema_hash == EXPECTED_GRAPH_SCHEMA_SHA256
        ),
        'manifest_has_148_subjects': len(multimodal_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': multimodal_df['subject_id'].is_unique,
        'target_policy_subjects_unique': (
            target_policy_df['subject_id'].is_unique
        ),
        'fold_manifest_subjects_unique': fold_df['subject_id'].is_unique,
        'all_three_manifest_subject_sets_match': (
            target_policy_subjects
            == fold_manifest_subjects
            == multimodal_subjects
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'development_fold_counts_match': fold_counts == EXPECTED_FOLD_COUNTS,
        'fivefold_loader_preflight_passed': (
            fivefold_gate.get('preflight_passed') is True
        ),
        'fivefold_preflight_locked_validation_targets_sealed': (
            fivefold_gate.get('locked_validation_targets_read') is False
        ),
        'fivefold_preflight_test_targets_sealed': (
            fivefold_gate.get('test_targets_read') is False
        ),
        'multimodal_loader_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_loader_locked_validation_targets_sealed': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_loader_test_targets_sealed': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'v22_model_preflight_passed': model_gate.get('preflight_passed') is True,
        'v22_parameter_count_matches_r5': (
            int(model_gate.get('total_parameters', -1))
            == EXPECTED_R5_PARAMETERS
        ),
        'r5_pilot_complete': r5_gate.get('pilot_complete') is True,
        'r5_pilot_version_matches': (
            r5_gate.get('pilot_version') == R5_PILOT_VERSION
        ),
        'r5_code_revision_matches': (
            r5_gate.get('code_revision') == R5_CODE_REVISION
        ),
        'r5_checkpoint_path_matches': (
            str(r5_gate.get('best_checkpoint_path'))
            == str(R5_BEST_CHECKPOINT_PATH)
        ),
        'r5_checkpoint_hash_matches_gate': (
            r5_gate.get('best_checkpoint_sha256')
            == r5_checkpoint_hash_before
        ),
        'r5_best_epoch_is_in_history': r5_history_best_present,
        'r5_summary_schema_complete': (
            r5_summary_required.issubset(r5_summary)
        ),
        'r5_gate_and_epoch_summary_match': all(
            close(metric_value(r5_summary, name), metric_value(r5_gate_summary, name))
            for name in [
                'positive_mean_dice', 'positive_median_dice',
                'positive_global_dice', 'small_case_mean_dice',
                'small_component_recall'
            ]
        ),
        'r7_readonly_r5_recompute_is_within_cross_contract_tolerance': bool(
            cross_contract_df.loc[
                cross_contract_df['comparison'].str.startswith('r7_'),
                'matches',
            ].all()
        ),
        'r9_baseline_provenance_is_reconciled': bool(
            cross_contract_df.loc[
                cross_contract_df['comparison'].str.startswith('r9_baseline_'),
                'matches',
            ].all()
        ),
        'all_required_cross_contract_comparisons_match': bool(
            cross_contract_df.loc[
                cross_contract_df['required_for_authorization'], 'matches'
            ].all()
        ),
        'case_metric_subjects_are_exactly_fold0_heldout': (
            fold0_case_subjects_match
        ),
        'case_metric_subjects_are_unique': case_df['subject_id'].is_unique,
        'case_metric_row_count_is_20': len(case_df) == 20,
        'case_metrics_contain_no_locked_validation_subjects': (
            no_locked_subjects_in_metrics
        ),
        'case_metrics_contain_no_test_subjects': no_test_subjects_in_metrics,
        'component_intersections_reconstruct_to_integer': (
            intersections_reconstruct_to_integer
        ),
        'all_recomputed_metrics_match_stored_metrics': bool(
            reconciliation_df['matches'].all()
        ),
        'all_reconciled_values_are_finite': bool(np.isfinite(
            reconciliation_df[
                ['recomputed_value', 'stored_value', 'absolute_error']
            ].to_numpy(dtype=np.float64)
        ).all()),
        'small_component_denominator_is_positive': (
            small_component_denominator > 0
        ),
        'historical_0p0579_is_subject_weighted_small': close(
            r5_reported_subject_weighted_small,
            small_subject_weighted,
            tolerance=CANONICAL_RECONCILIATION_TOLERANCE,
        ),
        'historical_0p0347_is_equal_bin_macro_small': close(
            r9_reported_bin_macro,
            small_bin_macro,
            tolerance=CANONICAL_RECONCILIATION_TOLERANCE,
        ),
        'small_metric_definitions_are_distinct': not close(
            small_subject_weighted, small_bin_macro, tolerance=1e-4
        ),
        'r9_pilot_complete': r9_gate.get('pilot_complete') is True,
        'r9_has_no_eligible_checkpoint': (
            r9_gate.get('eligible_r9_checkpoint_found') is False
        ),
        'r9_directional_success_is_false': (
            r9_gate.get('directional_success') is False
        ),
        'r9_deployment_checkpoint_is_r5': (
            str(r9_gate.get('deployment_checkpoint'))
            == str(R5_BEST_CHECKPOINT_PATH)
        ),
        'r9_rejected_candidate_is_not_mislabeled_as_deployment': (
            len(r9_selected) > 0
            and r9_gate.get('directional_success') is False
            and str(r9_gate.get('deployment_checkpoint'))
            == str(R5_BEST_CHECKPOINT_PATH)
        ),
        'r7_is_not_selected': (
            r7_gate.get('pilot_complete') is True
            and r7_gate.get('r7_directional_success_passed') is False
            and
            architecture_df.loc[
                architecture_df['experiment'] == 'R7',
                'eligible_for_selection',
            ].eq(False).all()
        ),
        'r8_incomplete_or_completed_rejected_and_not_selected': (
            (not r8_complete or not r8_accepted)
            and architecture_df.loc[
                architecture_df['experiment'] == 'R8',
                'eligible_for_selection',
            ].eq(False).all()
        ),
        'r8_completed_result_cannot_authorize_replication': (
            not r8_complete
            or (
                r8_result.get('directional_success') is False
                and r8_result.get('fold_replication_preflight_allowed')
                is False
            )
        ),
        'r8_history_reaches_reported_completed_epoch': (
            r8_history_rows > 0
            and r8_last_epoch
            == int(r8_result.get('last_completed_epoch', -1))
        ),
        'r8_final_summary_hash_matches_gate': (
            r8_result.get('final_summary_sha256')
            == file_sha256(R8_FINAL_SUMMARY_PATH)
        ),
        'r9_is_not_selected': (
            architecture_df.loc[
                architecture_df['experiment'] == 'R9',
                'eligible_for_selection',
            ].eq(False).all()
        ),
        'r9_final_summary_hash_matches_gate': (
            r9_gate.get('final_summary_sha256')
            == file_sha256(R9_FINAL_SUMMARY_PATH)
        ),
        'r9_summary_and_gate_selection_agree': (
            r9_summary.get('eligible_r9_checkpoint_found') is False
            and r9_summary.get('directional_success') is False
            and str(r9_summary.get('deployment_checkpoint'))
            == str(R5_BEST_CHECKPOINT_PATH)
        ),
        'r5_is_only_selected_architecture': (
            architecture_df['eligible_for_selection'].sum() == 1
            and architecture_df.loc[
                architecture_df['eligible_for_selection'], 'experiment'
            ].tolist() == ['R5']
        ),
        'metric_contract_uses_unambiguous_small_names': (
            metric_definitions['prohibited_ambiguous_label'] == 'small Dice'
            and metric_definitions['historical_discrepancy_resolution'][
                'values_are_not_contradictory'
            ] is True
        ),
        'architecture_and_hyperparameters_are_frozen': (
            authorization_policy['architecture_frozen'] is True
            and authorization_policy['hyperparameters_frozen_from_r5'] is True
        ),
        'only_folds_1_to_4_require_new_training': (
            authorization_policy['new_training_authorized_folds']
            == [1, 2, 3, 4]
        ),
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'target_arrays_not_read_by_this_cell': True,
        'model_not_constructed': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
    }

    
    
    r5_checkpoint_hash_after = file_sha256(R5_BEST_CHECKPOINT_PATH)
    checks['r5_checkpoint_unchanged'] = (
        r5_checkpoint_hash_after == r5_checkpoint_hash_before
    )

    checks_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in checks.items()
    ])
    authorization_passed = bool(all(checks.values()))

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_json(metric_definitions, METRIC_DEFINITIONS_PATH)
    atomic_write_csv(reconciliation_df, METRIC_RECONCILIATION_PATH)
    atomic_write_csv(architecture_df, ARCHITECTURE_SELECTION_PATH)
    atomic_write_csv(cross_contract_df, CROSS_CONTRACT_AUDIT_PATH)
    atomic_write_json(authorization_policy, AUTHORIZATION_POLICY_PATH)
    atomic_write_csv(checks_df, CHECKS_PATH)

    output_hashes = {
        path.name: file_sha256(path) for path in OUTPUT_ARTIFACTS
    }

    input_hashes = {
        str(TARGET_POLICY_MANIFEST_PATH): target_policy_hash,
        str(FOLD_MANIFEST_PATH): fold_manifest_hash,
        str(MULTIMODAL_MANIFEST_PATH): multimodal_manifest_hash,
        str(GRAPH_SCHEMA_PATH): graph_schema_hash,
        str(FIVEFOLD_GATE_PATH): file_sha256(FIVEFOLD_GATE_PATH),
        str(MULTIMODAL_GATE_PATH): file_sha256(MULTIMODAL_GATE_PATH),
        str(V22_MODEL_GATE_PATH): file_sha256(V22_MODEL_GATE_PATH),
        str(R5_RESULT_GATE_PATH): file_sha256(R5_RESULT_GATE_PATH),
        str(R5_BEST_CHECKPOINT_PATH): r5_checkpoint_hash_before,
        str(r5_case_metrics_path): file_sha256(r5_case_metrics_path),
        str(r5_tier_metrics_path): file_sha256(r5_tier_metrics_path),
        str(r5_validation_summary_path): file_sha256(
            r5_validation_summary_path
        ),
        str(R7_RESULT_GATE_PATH): file_sha256(R7_RESULT_GATE_PATH),
        str(R7_R5_BASELINE_DIAGNOSTIC_PATH): file_sha256(
            R7_R5_BASELINE_DIAGNOSTIC_PATH
        ),
        str(R8_RESULT_GATE_PATH): file_sha256(R8_RESULT_GATE_PATH),
        str(R8_HISTORY_PATH): file_sha256(R8_HISTORY_PATH),
        str(R8_FINAL_SUMMARY_PATH): file_sha256(R8_FINAL_SUMMARY_PATH),
        str(R9_RESULT_GATE_PATH): file_sha256(R9_RESULT_GATE_PATH),
        str(R9_FINAL_SUMMARY_PATH): file_sha256(R9_FINAL_SUMMARY_PATH),
    }

    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'reconciliation_version': RECONCILIATION_VERSION,
        'code_revision': CODE_REVISION,
        'metric_contract_revision': METRIC_CONTRACT_REVISION,
        'authorization_passed': bool(authorization_passed),
        'selected_architecture': 'R5',
        'selected_model_version': R5_MODEL_VERSION,
        'selected_pilot_version': R5_PILOT_VERSION,
        'selected_checkpoint_path': str(R5_BEST_CHECKPOINT_PATH),
        'selected_checkpoint_sha256': r5_checkpoint_hash_before,
        'selected_epoch': best_epoch,
        'canonical_metrics': {
            'positive_case_mean_dice': positive_mean,
            'positive_case_median_dice': positive_median,
            'pooled_global_dice': positive_global,
            'tiny_small_subject_weighted_mean_dice': small_subject_weighted,
            'tiny_small_bin_macro_mean_dice_secondary': small_bin_macro,
            'small_component_recall': small_component_recall,
        },
        'metric_counts': {
            'fold0_heldout_subjects': len(case_df),
            'positive_subjects': len(positive),
            'tiny_subjects': len(tiny_cases),
            'small_subjects': len(small_bin_cases),
            'tiny_small_subjects': len(small_cases),
            'small_components': small_component_denominator,
        },
        'historical_small_metric_discrepancy_resolved': True,
        'canonical_reconciliation_tolerance': (
            CANONICAL_RECONCILIATION_TOLERANCE
        ),
        'cross_contract_recomputation_tolerance': (
            CROSS_CONTRACT_RECOMPUTATION_TOLERANCE
        ),
        'r8_status': r8_status,
        'r8_complete': bool(r8_complete),
        'r8_directional_success': bool(r8_directional_success),
        'r8_fold_replication_preflight_allowed': bool(
            r8_replication_allowed
        ),
        'new_training_authorized_folds': [1, 2, 3, 4],
        'fold0_artifacts_reused': True,
        'development_fivefold_r5_training_authorized': bool(
            authorization_passed
        ),
        'full_fivefold_training_authorized': bool(authorization_passed),
        'locked_validation_evaluation_authorized': False,
        'test_evaluation_authorized': False,
        'architecture_changes_authorized': False,
        'hyperparameter_changes_authorized': False,
        'target_arrays_read': False,
        'target_arrays_read_splits': [],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'model_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'metric_definitions_path': str(METRIC_DEFINITIONS_PATH),
        'metric_reconciliation_path': str(METRIC_RECONCILIATION_PATH),
        'architecture_selection_path': str(ARCHITECTURE_SELECTION_PATH),
        'cross_contract_metric_audit_path': str(CROSS_CONTRACT_AUDIT_PATH),
        'authorization_policy_path': str(AUTHORIZATION_POLICY_PATH),
        'checks_path': str(CHECKS_PATH),
        'recommended_next_action': (
            'Build the isolated R5 fold-replication trainer for folds 1-4. '
            'Reuse completed fold 0; keep locked validation and test sealed.'
        ),
    }
    atomic_write_json(gate, AUTHORIZATION_GATE_PATH)

    print('\nCanonical metric reconciliation:')
    display(reconciliation_df)
    print('\nArchitecture selection:')
    display(architecture_df)
    print('\nCross-contract metric audit:')
    display(cross_contract_df)
    print('\nAuthorization checks:')
    display(checks_df)

    if not authorization_passed:
        failed = [name for name, passed in checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 R5 METRIC RECONCILIATION/AUTHORIZATION FAILED. Do not '
            'launch fold replication. Failed checks: ' + ', '.join(failed)
        )

    print('\n' + '=' * 80)
    print('TRIAL8 R5 METRIC RECONCILIATION AND FIVE-FOLD AUTHORIZATION PASSED')
    print('=' * 80)
    print(f'R5 best epoch:                      {best_epoch}')
    print(f'R5 mean Dice:                       {positive_mean:.4f}')
    print(f'R5 global Dice:                     {positive_global:.4f}')
    print('Canonical tiny/small subject mean: '
          f'{small_subject_weighted:.4f}')
    print('Secondary tiny/small bin macro:    '
          f'{small_bin_macro:.4f}')
    print(f'Small-component recall:             {small_component_recall:.4f}')
    print('Historical 0.0579 / 0.0347 conflict: resolved as different aggregations')
    print('Selected architecture:              R5 VascuFateNet V2.2')
    print('Fold 0:                              reuse completed artifacts')
    print('New training authorized folds:      1, 2, 3, 4')
    print('Architecture/hyperparameter changes: False / False')
    print('Target arrays read:                 False')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer steps:                    0')
    print(f'Selected checkpoint:                {R5_BEST_CHECKPOINT_PATH}')
    print(f'Authorization gate:                 {AUTHORIZATION_GATE_PATH}')
    print('\nNext: build the isolated R5 fold-replication trainer for folds 1-4.')


run_reconciliation()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()












import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r1_frozen_r5_development_fold_replication'
REPLICATION_SERIES_VERSION = 'v2_2_r5_fivefold_replication_v1'
AUTHORIZED_REPLICATION_FOLDS = (1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    gate_path = root / 'replication_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    gate = load_json(gate_path)
    return bool(
        gate.get('replication_complete') is True
        and gate.get('development_fold') == int(fold)
        and gate.get('code_revision') == PILOT_CODE_REVISION
        and gate.get('replication_series_version')
        == REPLICATION_SERIES_VERSION
        and gate.get('best_checkpoint_path') == str(best_path)
        and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
    )


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_replication_fold{REPLICATION_FOLD}_r1'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
EXPECTED_V22_PARAMETERS = 16_870_679
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'replication_result_gate.json'
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'active_fold_is_authorized': (
        REPLICATION_FOLD
        in authorization_gate.get('new_training_authorized_folds', [])
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if PILOT_RESULT_GATE_PATH.exists():
        completed = load_json(PILOT_RESULT_GATE_PATH)
        if (
            completed.get('replication_complete') is True
            and completed.get('pilot_version') == PILOT_VERSION
            and completed.get('code_revision') == PILOT_CODE_REVISION
            and completed.get('development_fold') == REPLICATION_FOLD
            and completed.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and completed.get('fresh_start_from_initialization') is True
            and completed.get('parent_checkpoint_migration_allowed') is False
            and PILOT_BEST_CHECKPOINT_PATH.exists()
            and completed.get('best_checkpoint_sha256')
            == file_sha256(PILOT_BEST_CHECKPOINT_PATH)
        ):
            print('\n' + '=' * 80)
            print(
                f'TRIAL8 V2.2 R5 FOLD-{REPLICATION_FOLD} '
                'REPLICATION ALREADY COMPLETE'
            )
            print('=' * 80)
            print(f'Best epoch:                         {completed["best_epoch"]}')
            print(f'Best pilot score:                   {completed["best_score"]:.4f}')
            print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
            print('No model checkpoint or training history was rewritten.')
            return

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    best_score = -float('inf')
    best_epoch = 0
    best_threshold = 0.50
    best_proposal_threshold = 0.10
    validations_without_improvement = 0
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        update_step = int(checkpoint['update_step'])
        best_score = float(checkpoint['best_score'])
        best_epoch = int(checkpoint['best_epoch'])
        best_threshold = float(checkpoint['best_threshold'])
        best_proposal_threshold = float(checkpoint['best_proposal_threshold'])
        validations_without_improvement = int(
            checkpoint['validations_without_improvement']
        )
        if PILOT_HISTORY_PATH.exists():
            history_rows = pd.read_csv(PILOT_HISTORY_PATH).to_dict('records')
    else:
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} replication checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 DEVELOPMENT FOLD-{REPLICATION_FOLD} '
        'REPLICATION STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Development folds 1-4 authorized:    True')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')

    stop_reason = 'maximum_epochs'
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        improved = current_score > best_score + EARLY_STOPPING_MINIMUM_DELTA
        if improved:
            best_score = current_score
            best_epoch = epoch
            best_threshold = float(summary['threshold'])
            best_proposal_threshold = float(summary['proposal_threshold'])
            validations_without_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_improvement = 0
        else:
            validations_without_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'best_score': best_score, 'best_epoch': best_epoch,
            'improved': improved,
            'validations_without_improvement': validations_without_improvement,
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'best_score': best_score, 'best_epoch': best_epoch,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'best_epoch': best_epoch, 'best_score': best_score,
            'best_threshold': best_threshold,
            'best_proposal_threshold': best_proposal_threshold,
            'validations_without_improvement': validations_without_improvement,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, best={best_score:.4f} '
            f'at epoch {best_epoch}, patience='
            f'{validations_without_improvement}/{EARLY_STOPPING_PATIENCE}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_improvement >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH, map_location=device, weights_only=False
    )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    best_epoch = int(best_checkpoint['best_epoch'])
    best_summary = dict(best_checkpoint['validation_summary'])
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'pilot_complete': True,
        'replication_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'best_epoch': best_epoch,
        'best_score': float(best_checkpoint['best_score']),
        'best_threshold': float(best_checkpoint['best_threshold']),
        'best_proposal_threshold': float(
            best_checkpoint['best_proposal_threshold']
        ),
        'best_validation_summary': best_summary,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'architecture_or_hyperparameter_changes_made': False,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
    }, PILOT_STATE_PATH)
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 DEVELOPMENT FOLD-{REPLICATION_FOLD} '
        'REPLICATION COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Development folds 1-4 authorized:    True')
    print('Architecture/hyperparameters changed: False')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')


run_v22_replication_fold()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )


run_v22_replication_fold()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )


run_v22_replication_fold()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )


run_v22_replication_fold()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )


run_v22_replication_fold()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )


run_v22_replication_fold()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )


run_v22_replication_fold()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            pip_name or import_name,
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')

from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
TRAINING_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
TRAINING_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
AGGREGATION_VERSION = 'v1_final_fivefold_oof_threshold_freeze'
AGGREGATION_CODE_REVISION = 'r1_exact_sufficient_statistics_locked_policy'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_PARAMETERS = 16_870_679
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_AUTHORIZATION_GATE_VERSION = (
    'v1_r5_metric_reconciliation_fivefold_authorization'
)
EXPECTED_FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
BOOTSTRAP_REPLICATES = 10_000
BOOTSTRAP_SEED = 20261011
FLOAT_ATOL = 1.0e-8

FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
AUTHORIZATION_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / EXPECTED_AUTHORIZATION_GATE_VERSION
    / 'r5_fivefold_evaluation_authorization_gate.json'
)
TRAINING_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION / TRAINING_SERIES_VERSION
)
FINAL_SERIES_STATUS_PATH = (
    TRAINING_SERIES_ROOT / 'final_fivefold_status.json'
)

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / AGGREGATION_VERSION
)
FOLD_INTEGRITY_AUDIT_PATH = OUTPUT_ROOT / 'fold_integrity_audit.csv'
OOF_SUBJECT_AUDIT_PATH = OUTPUT_ROOT / 'oof_subject_audit.csv'
POOLED_LOCAL_CASE_PATH = (
    OUTPUT_ROOT / 'pooled_local_threshold_case_sufficient_statistics.csv'
)
POOLED_PROPOSAL_CASE_PATH = (
    OUTPUT_ROOT / 'pooled_proposal_threshold_case_sufficient_statistics.csv'
)
LOCAL_THRESHOLD_SUMMARY_PATH = (
    OUTPUT_ROOT / 'pooled_local_threshold_summary.csv'
)
PROPOSAL_THRESHOLD_SUMMARY_PATH = (
    OUTPUT_ROOT / 'pooled_proposal_threshold_summary.csv'
)
SELECTED_OOF_CASE_PATH = (
    OUTPUT_ROOT / 'oof_case_metrics_at_frozen_thresholds.csv'
)
FOLD_METRICS_PATH = OUTPUT_ROOT / 'fold_metrics_at_frozen_threshold.csv'
FOLD_VARIABILITY_PATH = OUTPUT_ROOT / 'fold_variability_summary.csv'
LESION_BIN_METRICS_PATH = OUTPUT_ROOT / 'lesion_bin_metrics.csv'
BOOTSTRAP_CI_PATH = OUTPUT_ROOT / 'oof_bootstrap_confidence_intervals.csv'
SOURCE_MANIFEST_PATH = OUTPUT_ROOT / 'source_artifact_manifest.csv'
DEPLOYMENT_POLICY_PATH = OUTPUT_ROOT / 'frozen_deployment_policy.json'
AGGREGATION_GATE_PATH = OUTPUT_ROOT / 'oof_threshold_freeze_gate.json'

EXPECTED_FOLD_ARTIFACTS = {
    'validation_summary', 'case_metrics', 'component_metrics',
    'component_tier_metrics', 'local_threshold_summary',
    'local_threshold_case_metrics', 'proposal_threshold_summary',
    'proposal_threshold_case_metrics', 'branch_reliance',
    'branch_reliance_history', 'fit_target_access_audit',
    'fit_component_audit', 'epoch1_schedule_audit',
    'epoch1_negative_skip_audit',
}

REQUIRED_CASE_COLUMNS = {
    'development_fold', 'epoch', 'subject_id', 'probability_source',
    'threshold', 'lesion_volume_bin', 'hard_metric_eligible',
    'target_voxels', 'predicted_voxels', 'intersection_voxels',
    'case_dice', 'target_volume_ml', 'predicted_volume_ml',
    'lesion_true_positive', 'lesion_false_positive',
    'lesion_false_negative', 'component_true_positive',
    'component_total', 'small_component_true_positive',
    'small_component_total',
}






_SHA256_CACHE = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    cache_key = (str(path.resolve()), int(stat.st_size), int(stat.st_mtime_ns))
    if cache_key in _SHA256_CACHE:
        return _SHA256_CACHE[cache_key]
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    result = digest.hexdigest()
    _SHA256_CACHE[cache_key] = result
    return result


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def to_builtin(value):
    if isinstance(value, dict):
        return {str(key): to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n'}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def subject_hash(subjects):
    payload = '\n'.join(sorted(str(value) for value in subjects)).encode(
        'utf-8'
    )
    return hashlib.sha256(payload).hexdigest()


def finite_float(value, label):
    result = float(value)
    if not math.isfinite(result):
        raise RuntimeError(f'{label} is not finite: {value!r}')
    return result


def floats_match(left, right, atol=FLOAT_ATOL):
    try:
        return bool(np.isclose(
            float(left), float(right), rtol=0.0, atol=float(atol)
        ))
    except (TypeError, ValueError):
        return False


def require_inside(path_value, root, label):
    path = Path(str(path_value))
    if not path.is_absolute():
        raise RuntimeError(f'{label} is not an absolute path: {path}')
    if not path.exists() or not path.is_file():
        raise RuntimeError(f'{label} does not exist: {path}')
    resolved_root = Path(root).resolve()
    resolved_path = path.resolve()
    try:
        resolved_path.relative_to(resolved_root)
    except ValueError as error:
        raise RuntimeError(
            f'{label} escapes its authorized root: {path}'
        ) from error
    return path


def checks_or_raise(checks, prefix):
    failed = [name for name, passed in checks.items() if not bool(passed)]
    if failed:
        raise RuntimeError(prefix + ': ' + ', '.join(failed))


def source_record(path, role, fold=None):
    path = Path(path)
    stat = path.stat()
    return {
        'role': str(role),
        'development_fold': None if fold is None else int(fold),
        'path': str(path),
        'size_bytes': int(stat.st_size),
        'mtime_ns_before': int(stat.st_mtime_ns),
        'sha256': file_sha256(path),
    }


def source_stats_unchanged(source_records):
    checks = {}
    for index, record in enumerate(source_records):
        path = Path(record['path'])
        stat = path.stat()
        key = f'{index:03d}_{record["role"]}'
        checks[key] = bool(
            int(stat.st_size) == int(record['size_bytes'])
            and int(stat.st_mtime_ns) == int(record['mtime_ns_before'])
        )
    return checks






def validate_case_frame(
    frame, fold, epoch, expected_subjects, thresholds, probability_source,
):
    missing_columns = sorted(REQUIRED_CASE_COLUMNS - set(frame.columns))
    if missing_columns:
        raise RuntimeError(
            f'Fold-{fold} {probability_source} case statistics are missing: '
            + ', '.join(missing_columns)
        )
    frame = frame.copy()
    frame['subject_id'] = frame['subject_id'].astype(str)
    frame['hard_metric_eligible'] = frame[
        'hard_metric_eligible'
    ].map(as_bool)
    integer_columns = [
        'development_fold', 'epoch', 'target_voxels', 'predicted_voxels',
        'intersection_voxels', 'lesion_true_positive',
        'lesion_false_positive', 'lesion_false_negative',
        'component_true_positive', 'component_total',
        'small_component_true_positive', 'small_component_total',
    ]
    for column in integer_columns:
        numeric = pd.to_numeric(frame[column], errors='raise')
        if not np.allclose(numeric, np.round(numeric), rtol=0.0, atol=0.0):
            raise RuntimeError(
                f'Fold-{fold} {probability_source} column {column} is not integer.'
            )
        frame[column] = numeric.astype(np.int64)
    for column in [
        'threshold', 'case_dice', 'target_volume_ml', 'predicted_volume_ml',
    ]:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    nonfinite_columns = ['threshold', 'target_volume_ml', 'predicted_volume_ml']
    if not np.isfinite(frame[nonfinite_columns].to_numpy(dtype=float)).all():
        raise RuntimeError(
            f'Fold-{fold} {probability_source} statistics contain nonfinite values.'
        )
    observed_thresholds = np.sort(frame['threshold'].unique())
    expected_thresholds = np.asarray(thresholds, dtype=float)
    expected_subject_set = set(str(value) for value in expected_subjects)
    key_duplicates = frame.duplicated(['subject_id', 'threshold']).any()
    per_subject_threshold_counts = frame.groupby('subject_id')[
        'threshold'
    ].nunique()
    basic_checks = {
        'fold_exact': set(frame['development_fold'].tolist()) == {int(fold)},
        'epoch_exact': set(frame['epoch'].tolist()) == {int(epoch)},
        'probability_source_exact': (
            set(frame['probability_source'].astype(str).tolist())
            == {str(probability_source)}
        ),
        'subjects_exact': set(frame['subject_id']) == expected_subject_set,
        'subject_count_exact': frame['subject_id'].nunique()
        == len(expected_subject_set),
        'row_count_exact': len(frame)
        == len(expected_subject_set) * len(expected_thresholds),
        'keys_unique': not bool(key_duplicates),
        'threshold_grid_exact': (
            observed_thresholds.shape == expected_thresholds.shape
            and np.allclose(
                observed_thresholds, expected_thresholds,
                rtol=0.0, atol=FLOAT_ATOL,
            )
        ),
        'every_subject_has_full_grid': bool(
            (per_subject_threshold_counts == len(expected_thresholds)).all()
        ),
        'counts_nonnegative': bool((frame[integer_columns[2:]] >= 0).all().all()),
        'intersection_bounded': bool((
            frame['intersection_voxels']
            <= np.minimum(frame['target_voxels'], frame['predicted_voxels'])
        ).all()),
        'component_tp_bounded': bool((
            frame['component_true_positive'] <= frame['component_total']
        ).all()),
        'small_component_tp_bounded': bool((
            frame['small_component_true_positive']
            <= frame['small_component_total']
        ).all()),
        'volumes_nonnegative': bool((
            frame[['target_volume_ml', 'predicted_volume_ml']] >= 0
        ).all().all()),
    }
    checks_or_raise(
        basic_checks,
        f'Fold-{fold} {probability_source} sufficient-statistics audit failed',
    )

    invariant_columns = [
        'lesion_volume_bin', 'hard_metric_eligible', 'target_voxels',
        'target_volume_ml', 'component_total', 'small_component_total',
    ]
    invariants = frame.groupby('subject_id')[invariant_columns].nunique(
        dropna=False
    )
    if not bool((invariants == 1).all().all()):
        raise RuntimeError(
            f'Fold-{fold} {probability_source} target metadata changes across '
            'thresholds.'
        )

    positive = frame[
        frame['hard_metric_eligible'] & (frame['target_voxels'] > 0)
    ].copy()
    expected_dice = (
        2.0 * positive['intersection_voxels'].to_numpy(dtype=float)
        / np.maximum(
            positive['predicted_voxels'].to_numpy(dtype=float)
            + positive['target_voxels'].to_numpy(dtype=float),
            1.0,
        )
    )
    observed_dice = positive['case_dice'].to_numpy(dtype=float)
    if not np.allclose(
        observed_dice, expected_dice, rtol=0.0, atol=FLOAT_ATOL,
    ):
        raise RuntimeError(
            f'Fold-{fold} {probability_source} stored Dice does not match '
            'voxel sufficient statistics.'
        )
    return frame.sort_values(['subject_id', 'threshold']).reset_index(drop=True)


def aggregate_case_metrics(frame):
    frame = frame.copy()
    if 'hard_metric_eligible' not in frame.columns:
        raise RuntimeError('Metric frame lacks hard_metric_eligible.')
    frame['hard_metric_eligible'] = frame['hard_metric_eligible'].map(as_bool)
    hard = frame['hard_metric_eligible'].to_numpy(dtype=bool)
    target = frame['target_voxels'].to_numpy(dtype=np.int64)
    predicted = frame['predicted_voxels'].to_numpy(dtype=np.int64)
    intersection = frame['intersection_voxels'].to_numpy(dtype=np.int64)
    positive = hard & (target > 0)
    empty = hard & (target == 0)
    dice = np.zeros(len(frame), dtype=np.float64)
    dice[positive] = (
        2.0 * intersection[positive]
        / np.maximum(predicted[positive] + target[positive], 1)
    )
    bins = frame['lesion_volume_bin'].astype(str).to_numpy()
    tiny_small = positive & np.isin(bins, ['tiny', 'small'])

    lesion_tp = int(frame.loc[hard, 'lesion_true_positive'].sum())
    lesion_fp = int(frame.loc[hard, 'lesion_false_positive'].sum())
    lesion_fn = int(frame.loc[hard, 'lesion_false_negative'].sum())
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )

    component_tp = int(frame.loc[positive, 'component_true_positive'].sum())
    component_total = int(frame.loc[positive, 'component_total'].sum())
    small_tp = int(
        frame.loc[positive, 'small_component_true_positive'].sum()
    )
    small_total = int(frame.loc[positive, 'small_component_total'].sum())
    positive_dice = dice[positive]
    small_dice = dice[tiny_small]
    global_dice = (
        2.0 * int(intersection[positive].sum())
        / max(int(predicted[positive].sum() + target[positive].sum()), 1)
    )
    positive_mean = float(positive_dice.mean()) if positive_dice.size else 0.0
    positive_median = (
        float(np.median(positive_dice)) if positive_dice.size else 0.0
    )
    small_mean = float(small_dice.mean()) if small_dice.size else 0.0
    selection_score = (
        0.60 * positive_mean + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean
    )
    predicted_positive_volume = float(
        frame.loc[positive, 'predicted_volume_ml'].sum()
    )
    target_positive_volume = float(
        frame.loc[positive, 'target_volume_ml'].sum()
    )
    result = {
        'positive_mean_dice': positive_mean,
        'positive_median_dice': positive_median,
        'positive_global_dice': float(global_dice),
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_tp / max(component_total, 1)),
        'small_component_recall': float(small_tp / max(small_total, 1)),
        'small_case_mean_dice': small_mean,
        'selection_score': float(selection_score),
        'positive_subjects': int(positive.sum()),
        'hard_eligible_subjects': int(hard.sum()),
        'hard_ineligible_subjects': int((~hard).sum()),
        'empty_hard_eligible_subjects': int(empty.sum()),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
        'lesion_true_positive': lesion_tp,
        'lesion_false_positive': lesion_fp,
        'lesion_false_negative': lesion_fn,
        'total_intersection_voxels': int(intersection[positive].sum()),
        'total_predicted_voxels': int(predicted[positive].sum()),
        'total_target_voxels': int(target[positive].sum()),
        'predicted_to_target_volume_ratio': float(
            predicted_positive_volume / max(target_positive_volume, 1e-12)
        ),
        'mean_empty_false_positive_volume_ml': float(
            frame.loc[empty, 'predicted_volume_ml'].mean()
            if bool(empty.any()) else 0.0
        ),
        'maximum_empty_false_positive_volume_ml': float(
            frame.loc[empty, 'predicted_volume_ml'].max()
            if bool(empty.any()) else 0.0
        ),
    }
    for lesion_bin in ['tiny', 'small', 'medium', 'large', 'very_large']:
        mask = positive & (bins == lesion_bin)
        result[f'{lesion_bin}_subjects'] = int(mask.sum())
        result[f'{lesion_bin}_mean_dice'] = (
            float(dice[mask].mean()) if bool(mask.any()) else 0.0
        )
    tiny_value = result['tiny_mean_dice']
    small_value = result['small_mean_dice']
    result['tiny_small_bin_macro_mean_dice'] = float(
        0.5 * (tiny_value + small_value)
    )
    return result


def summarize_threshold_grid(frame, thresholds, source):
    rows = []
    for threshold in thresholds:
        selected = frame[np.isclose(
            frame['threshold'].to_numpy(dtype=float), float(threshold),
            rtol=0.0, atol=FLOAT_ATOL,
        )].copy()
        summary = aggregate_case_metrics(selected)
        summary.update({
            'probability_source': str(source),
            'threshold': float(threshold),
            'oof_subjects': int(selected['subject_id'].nunique()),
            'development_folds': int(selected['development_fold'].nunique()),
        })
        if source == 'proposal_probability':
            summary['proposal_selection_score'] = float(
                0.45 * summary['positive_global_dice']
                + 0.25 * summary['lesion_f1']
                + 0.15 * summary['lesion_precision']
                + 0.15 * summary['component_recall']
            )
        rows.append(summary)
    result = pd.DataFrame(rows)
    if source == 'probability':
        ranked = result.sort_values(
            ['selection_score', 'positive_mean_dice', 'threshold'],
            ascending=[False, False, True],
        ).copy()
    else:
        ranked = result.sort_values(
            ['proposal_selection_score', 'component_recall', 'threshold'],
            ascending=[False, False, True],
        ).copy()
    rank_map = {
        float(row.threshold): rank
        for rank, row in enumerate(ranked.itertuples(index=False), start=1)
    }
    result['selection_rank'] = result['threshold'].map(rank_map).astype(int)
    return result.sort_values('threshold').reset_index(drop=True), ranked.iloc[0]


def compare_summary(observed, expected, keys, label, atol=FLOAT_ATOL):
    checks = {}
    for key in keys:
        checks[f'{label}_{key}'] = (
            key in expected and floats_match(observed[key], expected[key], atol)
        )
    checks_or_raise(checks, f'{label} summary reconciliation failed')
    return checks


def metric_self_test():
    frame = pd.DataFrame([
        {
            'subject_id': 'a', 'lesion_volume_bin': 'small',
            'hard_metric_eligible': True, 'target_voxels': 10,
            'predicted_voxels': 10, 'intersection_voxels': 5,
            'target_volume_ml': 1.0, 'predicted_volume_ml': 1.0,
            'lesion_true_positive': 1, 'lesion_false_positive': 0,
            'lesion_false_negative': 0, 'component_true_positive': 1,
            'component_total': 2, 'small_component_true_positive': 1,
            'small_component_total': 2,
        },
        {
            'subject_id': 'b', 'lesion_volume_bin': 'medium',
            'hard_metric_eligible': True, 'target_voxels': 20,
            'predicted_voxels': 20, 'intersection_voxels': 10,
            'target_volume_ml': 2.0, 'predicted_volume_ml': 2.0,
            'lesion_true_positive': 1, 'lesion_false_positive': 1,
            'lesion_false_negative': 0, 'component_true_positive': 1,
            'component_total': 1, 'small_component_true_positive': 0,
            'small_component_total': 0,
        },
        {
            'subject_id': 'c', 'lesion_volume_bin': 'empty',
            'hard_metric_eligible': True, 'target_voxels': 0,
            'predicted_voxels': 2, 'intersection_voxels': 0,
            'target_volume_ml': 0.0, 'predicted_volume_ml': 0.2,
            'lesion_true_positive': 0, 'lesion_false_positive': 1,
            'lesion_false_negative': 0, 'component_true_positive': 0,
            'component_total': 0, 'small_component_true_positive': 0,
            'small_component_total': 0,
        },
    ])
    summary = aggregate_case_metrics(frame)
    checks = {
        'positive_mean': floats_match(summary['positive_mean_dice'], 0.5),
        'positive_median': floats_match(summary['positive_median_dice'], 0.5),
        'global': floats_match(summary['positive_global_dice'], 0.5),
        'lesion_precision': floats_match(summary['lesion_precision'], 2 / 4),
        'lesion_recall': floats_match(summary['lesion_recall'], 1.0),
        'component_recall': floats_match(summary['component_recall'], 2 / 3),
        'small_component_recall': floats_match(
            summary['small_component_recall'], 0.5
        ),
        'empty_fp': floats_match(
            summary['mean_empty_false_positive_volume_ml'], 0.2
        ),
    }
    checks_or_raise(checks, 'Internal sufficient-statistics self-test failed')
    return checks






def bootstrap_confidence_intervals(frame):
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    frame = frame.reset_index(drop=True).copy()
    metrics = [
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'small_case_mean_dice', 'tiny_small_bin_macro_mean_dice',
        'predicted_to_target_volume_ratio',
    ]
    point = aggregate_case_metrics(frame)
    samples = {metric: np.zeros(BOOTSTRAP_REPLICATES) for metric in metrics}
    subject_count = len(frame)
    for replicate in range(BOOTSTRAP_REPLICATES):
        indices = rng.integers(0, subject_count, size=subject_count)
        summary = aggregate_case_metrics(frame.iloc[indices])
        for metric in metrics:
            samples[metric][replicate] = float(summary[metric])
    rows = []
    for metric in metrics:
        values = samples[metric]
        rows.append({
            'metric': metric,
            'point_estimate': float(point[metric]),
            'ci_95_lower': float(np.quantile(values, 0.025)),
            'ci_95_upper': float(np.quantile(values, 0.975)),
            'bootstrap_standard_error': float(values.std(ddof=1)),
            'bootstrap_replicates': BOOTSTRAP_REPLICATES,
            'bootstrap_seed': BOOTSTRAP_SEED,
            'method': 'subject_level_nonparametric_percentile_fixed_threshold',
        })
    return pd.DataFrame(rows)


def lesion_bin_metrics(frame):
    rows = []
    hard = frame['hard_metric_eligible'].map(as_bool)
    positive = frame[hard & (frame['target_voxels'] > 0)].copy()
    for lesion_bin in ['tiny', 'small', 'medium', 'large', 'very_large']:
        group = positive[
            positive['lesion_volume_bin'].astype(str) == lesion_bin
        ].copy()
        if group.empty:
            rows.append({
                'lesion_volume_bin': lesion_bin, 'subjects': 0,
                'mean_dice': 0.0, 'median_dice': 0.0,
                'pooled_global_dice': 0.0,
                'mean_target_volume_ml': 0.0,
                'mean_predicted_volume_ml': 0.0,
            })
            continue
        dice = (
            2.0 * group['intersection_voxels'].to_numpy(dtype=float)
            / np.maximum(
                group['predicted_voxels'].to_numpy(dtype=float)
                + group['target_voxels'].to_numpy(dtype=float),
                1.0,
            )
        )
        rows.append({
            'lesion_volume_bin': lesion_bin,
            'subjects': len(group),
            'mean_dice': float(dice.mean()),
            'median_dice': float(np.median(dice)),
            'pooled_global_dice': float(
                2.0 * group['intersection_voxels'].sum()
                / max(
                    int(group['predicted_voxels'].sum())
                    + int(group['target_voxels'].sum()),
                    1,
                )
            ),
            'mean_target_volume_ml': float(group['target_volume_ml'].mean()),
            'mean_predicted_volume_ml': float(
                group['predicted_volume_ml'].mean()
            ),
        })
    return pd.DataFrame(rows)






def validate_existing_result():
    if not AGGREGATION_GATE_PATH.exists():
        return None
    gate = load_json(AGGREGATION_GATE_PATH)
    checks = {
        'aggregation_complete': gate.get('aggregation_complete') is True,
        'threshold_frozen': gate.get('deployment_threshold_frozen') is True,
        'version_matches': gate.get('aggregation_version')
        == AGGREGATION_VERSION,
        'code_revision_matches': gate.get('code_revision')
        == AGGREGATION_CODE_REVISION,
        'training_series_matches': gate.get('training_series_version')
        == TRAINING_SERIES_VERSION,
        'all_102_oof_subjects': int(gate.get('oof_subjects', -1)) == 102,
        'folds_exact': tuple(gate.get('development_folds', []))
        == EXPECTED_FOLDS,
        'locked_validation_targets_not_read':
        gate.get('locked_validation_targets_read') is False,
        'test_targets_not_read': gate.get('test_targets_read') is False,
        'model_not_constructed': gate.get('model_constructed') is False,
        'optimizer_not_constructed': gate.get('optimizer_constructed') is False,
        'training_not_performed': gate.get('training_performed') is False,
    }
    output_manifest = gate.get('output_artifact_manifest', {})
    source_manifest = gate.get('source_artifact_manifest', [])
    checks['output_manifest_nonempty'] = bool(output_manifest)
    checks['source_manifest_nonempty'] = bool(source_manifest)
    if isinstance(output_manifest, dict):
        for name, record in output_manifest.items():
            try:
                path = require_inside(
                    record['path'], OUTPUT_ROOT, f'existing output {name}'
                )
                checks[f'output_{name}_hash'] = (
                    record.get('sha256') == file_sha256(path)
                )
            except Exception:
                checks[f'output_{name}_hash'] = False
    if isinstance(source_manifest, list):
        for index, record in enumerate(source_manifest):
            try:
                path = Path(str(record['path']))
                checks[f'source_{index:03d}_hash'] = bool(
                    path.is_file()
                    and record.get('sha256') == file_sha256(path)
                )
            except Exception:
                checks[f'source_{index:03d}_hash'] = False
    checks_or_raise(
        checks,
        'Existing OOF threshold-freeze gate is present but invalid; preserve '
        'the directory and diagnose it rather than overwriting it',
    )
    return gate






def run_oof_aggregation():
    existing = validate_existing_result()
    if existing is not None:
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL FIVE-FOLD OOF THRESHOLD FREEZE ALREADY COMPLETE')
        print('=' * 80)
        print(
            'Frozen local threshold:             '
            f'{float(existing["frozen_local_threshold"]):.2f}'
        )
        print(
            'OOF positive-case mean Dice:        '
            f'{float(existing["pooled_oof_metrics"]["positive_mean_dice"]):.4f}'
        )
        print(
            'OOF pooled/global Dice:             '
            f'{float(existing["pooled_oof_metrics"]["positive_global_dice"]):.4f}'
        )
        print(f'Aggregation gate:                  {AGGREGATION_GATE_PATH}')
        print('No training or source artifact was changed.')
        return

    metric_self_test()

    required_paths = [
        FOLD_MANIFEST_PATH, MULTIMODAL_MANIFEST_PATH,
        AUTHORIZATION_GATE_PATH, FINAL_SERIES_STATUS_PATH,
    ]
    missing = [str(path) for path in required_paths if not path.is_file()]
    if missing:
        raise RuntimeError(
            'Required final-study input is missing:\n' + '\n'.join(missing)
        )

    source_records = [
        source_record(FOLD_MANIFEST_PATH, 'fold_manifest'),
        source_record(MULTIMODAL_MANIFEST_PATH, 'multimodal_manifest'),
        source_record(AUTHORIZATION_GATE_PATH, 'fivefold_authorization_gate'),
        source_record(FINAL_SERIES_STATUS_PATH, 'final_series_status'),
    ]

    authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
    final_status = load_json(FINAL_SERIES_STATUS_PATH)
    upstream_checks = {
        'fold_manifest_hash_matches': file_sha256(FOLD_MANIFEST_PATH)
        == EXPECTED_FOLD_MANIFEST_SHA256,
        'multimodal_manifest_hash_matches': file_sha256(
            MULTIMODAL_MANIFEST_PATH
        ) == EXPECTED_MULTIMODAL_MANIFEST_SHA256,
        'authorization_passed': (
            authorization_gate.get('authorization_passed') is True
            and authorization_gate.get(
                'development_fivefold_r5_training_authorized'
            ) is True
            and authorization_gate.get(
                'full_fivefold_training_authorized'
            ) is True
        ),
        'authorization_selected_r5': authorization_gate.get(
            'selected_architecture'
        ) in {'R5 VascuFateNet V2.2', 'R5'},
        'authorization_architecture_frozen': authorization_gate.get(
            'architecture_changes_authorized'
        ) is False,
        'authorization_hyperparameters_frozen': authorization_gate.get(
            'hyperparameter_changes_authorized'
        ) is False,
        'authorization_raw_targets_unread': authorization_gate.get(
            'target_arrays_read'
        ) is False,
        'authorization_locked_validation_targets_sealed': authorization_gate.get(
            'locked_validation_targets_read'
        ) is False,
        'authorization_test_targets_sealed': authorization_gate.get(
            'test_targets_read'
        ) is False,
        'authorization_training_not_performed': authorization_gate.get(
            'training_performed'
        ) is False,
        'series_complete': final_status.get('series_complete') is True,
        'series_version_exact': final_status.get(
            'replication_series_version'
        ) == TRAINING_SERIES_VERSION,
        'series_code_revision_exact': final_status.get('code_revision')
        == TRAINING_CODE_REVISION,
        'series_protocol_role_exact': final_status.get('final_protocol_role')
        == EXPECTED_FINAL_PROTOCOL_ROLE,
        'series_completed_folds_exact': tuple(final_status.get(
            'completed_folds', []
        )) == EXPECTED_FOLDS,
        'series_remaining_folds_empty': final_status.get(
            'remaining_folds', []
        ) == [],
        'series_oof_subjects_102': int(final_status.get(
            'completed_heldout_subjects', -1
        )) == 102,
        'series_strict_best_verified': final_status.get(
            'all_strict_numeric_best_verified'
        ) is True,
        'series_locked_validation_targets_sealed': final_status.get(
            'locked_validation_targets_read'
        ) is False,
        'series_test_targets_sealed': final_status.get(
            'test_targets_read'
        ) is False,
        'series_integrity_checks_all_pass': bool(
            final_status.get('series_integrity_checks')
            and all(
                value is True for value in final_status[
                    'series_integrity_checks'
                ].values()
            )
        ),
    }
    checks_or_raise(upstream_checks, 'Final-series upstream contract failed')

    multimodal_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_multimodal = {'subject_id', 'dev_split'}
    required_fold = {'subject_id', 'frozen_dev_split', 'development_fold'}
    checks_or_raise({
        'multimodal_columns_present': required_multimodal
        <= set(multimodal_df.columns),
        'fold_columns_present': required_fold <= set(fold_df.columns),
    }, 'Manifest schema audit failed')
    multimodal_df['subject_id'] = multimodal_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    multimodal_df = multimodal_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    manifest_checks = {
        'manifest_has_148_subjects': len(multimodal_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': multimodal_df['subject_id'].nunique()
        == EXPECTED_SUBJECTS,
        'fold_manifest_has_148_subjects': len(fold_df) == EXPECTED_SUBJECTS,
        'fold_manifest_subjects_unique': fold_df['subject_id'].nunique()
        == EXPECTED_SUBJECTS,
        'manifest_subjects_match': multimodal_df['subject_id'].tolist()
        == fold_df['subject_id'].tolist(),
        'split_counts_match': multimodal_df['dev_split'].astype(str)
        .value_counts().to_dict() == EXPECTED_SPLITS,
        'frozen_splits_match': multimodal_df['dev_split'].astype(str).tolist()
        == fold_df['frozen_dev_split'].astype(str).tolist(),
    }
    checks_or_raise(manifest_checks, 'Manifest content audit failed')
    multimodal_df['development_fold'] = fold_df['development_fold'].astype(int)
    development_df = multimodal_df[
        multimodal_df['dev_split'].astype(str) == 'train'
    ].copy()
    locked_subjects = set(multimodal_df.loc[
        multimodal_df['dev_split'].astype(str).isin(['val', 'test']),
        'subject_id',
    ])
    fold_counts = development_df['development_fold'].value_counts().to_dict()
    if fold_counts != EXPECTED_FOLD_COUNTS:
        raise RuntimeError(
            f'Development fold counts changed: {fold_counts} '
            f'!= {EXPECTED_FOLD_COUNTS}.'
        )

    status_records = {
        int(record['development_fold']): record
        for record in final_status.get('fold_records', [])
    }
    if set(status_records) != set(EXPECTED_FOLDS):
        raise RuntimeError('Final series status does not contain exactly folds 0-4.')

    local_frames = []
    proposal_frames = []
    fold_integrity_rows = []
    checkpoint_records = []
    fold_gates = {}
    fold_reconciliation_checks = {}

    for fold in EXPECTED_FOLDS:
        fold_root = TRAINING_SERIES_ROOT / f'fold_{fold}'
        gate_path = fold_root / 'final_fold_result_gate.json'
        if not gate_path.is_file():
            raise RuntimeError(f'Fold-{fold} result gate is missing: {gate_path}')
        status_record = status_records[fold]
        if status_record.get('result_gate_path') != str(gate_path):
            raise RuntimeError(f'Fold-{fold} status points to the wrong gate.')
        if status_record.get('result_gate_sha256') != file_sha256(gate_path):
            raise RuntimeError(f'Fold-{fold} result-gate hash changed.')
        source_records.append(source_record(gate_path, 'fold_result_gate', fold))
        gate = load_json(gate_path)
        fold_gates[fold] = gate

        heldout_subjects = sorted(
            development_df.loc[
                development_df['development_fold'] == fold, 'subject_id'
            ].astype(str).tolist()
        )
        fit_subjects = sorted(
            development_df.loc[
                development_df['development_fold'] != fold, 'subject_id'
            ].astype(str).tolist()
        )
        gate_checks = {
            'complete': gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True,
            'fold_exact': int(gate.get('development_fold', -1)) == fold,
            'heldout_count_exact': int(gate.get('heldout_subjects', -1))
            == EXPECTED_FOLD_COUNTS[fold],
            'fit_count_exact': int(gate.get('fit_subjects', -1))
            == 102 - EXPECTED_FOLD_COUNTS[fold],
            'heldout_hash_exact': gate.get('heldout_subject_hash')
            == subject_hash(heldout_subjects),
            'fit_hash_exact': gate.get('fit_subject_hash')
            == subject_hash(fit_subjects),
            'parameter_count_exact': int(gate.get('parameter_count', -1))
            == EXPECTED_PARAMETERS,
            'pilot_version_exact': gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{fold}_r2',
            'code_revision_exact': gate.get('code_revision')
            == TRAINING_CODE_REVISION,
            'series_version_exact': gate.get('replication_series_version')
            == TRAINING_SERIES_VERSION,
            'strict_numeric_best': gate.get('strict_numeric_best_verified')
            is True,
            'checkpoint_policy_exact': gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum',
            'checkpoint_delta_zero': floats_match(
                gate.get('checkpoint_selection_minimum_delta'), 0.0
            ),
            'checkpoint_and_patience_independent': gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True,
            'fresh_initialization': gate.get('fresh_start_from_initialization')
            is True,
            'no_parent_migration': gate.get(
                'parent_checkpoint_migration_allowed'
            ) is False,
            'architecture_unchanged': gate.get('architecture_changed_from_r5')
            is False,
            'optimization_unchanged': gate.get(
                'optimization_loss_sampling_changed_from_r5'
            ) is False,
            'raw_locked_validation_targets_sealed': gate.get(
                'locked_validation_targets_read'
            ) is False,
            'raw_test_targets_sealed': gate.get('test_targets_read') is False,
            'oof_statistics_complete': gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True,
            'fold_threshold_not_deployment_threshold': gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True,
            'protocol_role_exact': gate.get('final_protocol_role')
            == EXPECTED_FINAL_PROTOCOL_ROLE,
            'target_splits_train_only': gate.get('target_arrays_read_splits')
            == [f'train_fold{fold}_fit', f'train_fold{fold}_heldout'],
            'series_status_checkpoint_path_matches': status_record.get(
                'best_checkpoint_path'
            ) == gate.get('best_checkpoint_path'),
            'series_status_checkpoint_hash_matches': status_record.get(
                'best_checkpoint_sha256'
            ) == gate.get('best_checkpoint_sha256'),
            'series_status_best_epoch_matches': int(status_record.get(
                'best_epoch', -1
            )) == int(gate.get('best_epoch', -2)),
            'series_status_best_score_matches': floats_match(
                status_record.get('best_score'), gate.get('best_score')
            ),
        }
        for check_group in [
            'strict_best_checks', 'best_oof_sufficient_statistics_checks',
            'target_access_checks', 'final_stop_checks',
        ]:
            values = gate.get(check_group, {})
            gate_checks[f'{check_group}_all_true'] = bool(
                values and all(value is True for value in values.values())
            )
        checks_or_raise(gate_checks, f'Fold-{fold} final result contract failed')

        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if set(artifact_manifest) != EXPECTED_FOLD_ARTIFACTS:
            raise RuntimeError(
                f'Fold-{fold} best artifact bundle does not match the frozen schema.'
            )
        for name, record in artifact_manifest.items():
            artifact_path = require_inside(
                record.get('path'), fold_root,
                f'fold-{fold} artifact {name}',
            )
            if record.get('sha256') != file_sha256(artifact_path):
                raise RuntimeError(
                    f'Fold-{fold} artifact hash changed: {name}'
                )
            source_records.append(source_record(
                artifact_path, f'fold_artifact_{name}', fold
            ))

        best_checkpoint_path = require_inside(
            gate.get('best_checkpoint_path'), fold_root,
            f'fold-{fold} best checkpoint',
        )
        last_checkpoint_path = require_inside(
            gate.get('last_checkpoint_path'), fold_root,
            f'fold-{fold} last checkpoint',
        )
        history_path = require_inside(
            gate.get('history_path'), fold_root, f'fold-{fold} history'
        )
        config_path = require_inside(
            gate.get('pilot_config_path'), fold_root, f'fold-{fold} config'
        )
        for role, path, expected_hash in [
            ('best_checkpoint', best_checkpoint_path,
             gate.get('best_checkpoint_sha256')),
            ('last_checkpoint', last_checkpoint_path,
             gate.get('last_checkpoint_sha256')),
            ('training_history', history_path, gate.get('history_sha256')),
            ('pilot_config', config_path, gate.get('pilot_config_sha256')),
        ]:
            observed_hash = file_sha256(path)
            if expected_hash != observed_hash:
                raise RuntimeError(f'Fold-{fold} {role} hash changed.')
            source_records.append(source_record(path, role, fold))
        checkpoint_records.append({
            'development_fold': fold,
            'best_checkpoint_path': str(best_checkpoint_path),
            'best_checkpoint_sha256': str(gate['best_checkpoint_sha256']),
            'best_epoch': int(gate['best_epoch']),
            'parameter_count': int(gate['parameter_count']),
            'weight_state': 'ema_model_state',
        })

        local_path = require_inside(
            gate['best_local_threshold_case_metrics_path'], fold_root,
            f'fold-{fold} local OOF case statistics',
        )
        proposal_path = require_inside(
            gate['best_proposal_threshold_case_metrics_path'], fold_root,
            f'fold-{fold} proposal OOF case statistics',
        )
        local_manifest_path = Path(
            artifact_manifest['local_threshold_case_metrics']['path']
        )
        proposal_manifest_path = Path(
            artifact_manifest['proposal_threshold_case_metrics']['path']
        )
        path_checks = {
            'local_path_matches_artifact_manifest': local_path
            == local_manifest_path,
            'proposal_path_matches_artifact_manifest': proposal_path
            == proposal_manifest_path,
            'local_dedicated_hash_matches': gate[
                'best_local_threshold_case_metrics_sha256'
            ] == file_sha256(local_path),
            'proposal_dedicated_hash_matches': gate[
                'best_proposal_threshold_case_metrics_sha256'
            ] == file_sha256(proposal_path),
        }
        checks_or_raise(path_checks, f'Fold-{fold} OOF path contract failed')

        local = validate_case_frame(
            pd.read_csv(local_path), fold, int(gate['best_epoch']),
            heldout_subjects, LOCAL_THRESHOLDS, 'probability',
        )
        proposal = validate_case_frame(
            pd.read_csv(proposal_path), fold, int(gate['best_epoch']),
            heldout_subjects, PROPOSAL_THRESHOLDS, 'proposal_probability',
        )

        local_metadata = local.sort_values('threshold').drop_duplicates(
            'subject_id'
        ).set_index('subject_id')
        proposal_metadata = proposal.sort_values('threshold').drop_duplicates(
            'subject_id'
        ).set_index('subject_id')
        for column in [
            'lesion_volume_bin', 'hard_metric_eligible', 'target_voxels',
            'target_volume_ml', 'component_total', 'small_component_total',
        ]:
            left = local_metadata.loc[heldout_subjects, column]
            right = proposal_metadata.loc[heldout_subjects, column]
            if pd.api.types.is_numeric_dtype(left):
                matches = np.allclose(
                    left.to_numpy(dtype=float), right.to_numpy(dtype=float),
                    rtol=0.0, atol=FLOAT_ATOL,
                )
            else:
                matches = left.astype(str).tolist() == right.astype(str).tolist()
            if not matches:
                raise RuntimeError(
                    f'Fold-{fold} local/proposal target metadata differs: {column}'
                )

        best_local_rows = local[np.isclose(
            local['threshold'].to_numpy(dtype=float),
            float(gate['best_threshold']), rtol=0.0, atol=FLOAT_ATOL,
        )]
        best_proposal_rows = proposal[np.isclose(
            proposal['threshold'].to_numpy(dtype=float),
            float(gate['best_proposal_threshold']),
            rtol=0.0, atol=FLOAT_ATOL,
        )]
        recomputed_local = aggregate_case_metrics(best_local_rows)
        recomputed_proposal = aggregate_case_metrics(best_proposal_rows)
        stored = gate['best_validation_summary']
        local_keys = [
            'positive_mean_dice', 'positive_median_dice',
            'positive_global_dice', 'lesion_precision', 'lesion_recall',
            'lesion_f1', 'component_recall', 'small_component_recall',
            'small_case_mean_dice',
        ]
        compare_summary(
            recomputed_local, stored, local_keys, f'fold_{fold}_local'
        )
        proposal_checks = {
            f'fold_{fold}_proposal_global_dice': floats_match(
                recomputed_proposal['positive_global_dice'],
                stored['proposal_global_dice'],
            ),
            f'fold_{fold}_proposal_component_recall': floats_match(
                recomputed_proposal['component_recall'],
                stored['proposal_component_recall'],
            ),
            f'fold_{fold}_checkpoint_score': floats_match(
                0.90 * recomputed_local['selection_score']
                + 0.10 * recomputed_proposal['positive_global_dice'],
                gate['best_score'],
            ),
        }
        checks_or_raise(
            proposal_checks, f'Fold-{fold} proposal/checkpoint reconciliation failed'
        )
        fold_reconciliation_checks.update(proposal_checks)

        local_frames.append(local)
        proposal_frames.append(proposal)
        fold_integrity_rows.append({
            'development_fold': fold,
            'fit_subjects': len(fit_subjects),
            'heldout_subjects': len(heldout_subjects),
            'best_epoch': int(gate['best_epoch']),
            'fold_selected_local_threshold': float(gate['best_threshold']),
            'fold_selected_proposal_threshold': float(
                gate['best_proposal_threshold']
            ),
            'fold_best_score': float(gate['best_score']),
            'fold_best_mean_dice': float(stored['positive_mean_dice']),
            'fold_best_global_dice': float(stored['positive_global_dice']),
            'strict_numeric_best_verified': True,
            'oof_rows_local': len(local),
            'oof_rows_proposal': len(proposal),
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_sha256': file_sha256(best_checkpoint_path),
        })

    pooled_local = pd.concat(local_frames, ignore_index=True)
    pooled_proposal = pd.concat(proposal_frames, ignore_index=True)
    oof_subjects = set(pooled_local['subject_id'].astype(str))
    proposal_subjects = set(pooled_proposal['subject_id'].astype(str))
    development_subjects = set(development_df['subject_id'].astype(str))
    oof_checks = {
        'local_oof_subjects_exactly_102': len(oof_subjects) == 102,
        'proposal_oof_subjects_exactly_102': len(proposal_subjects) == 102,
        'local_and_proposal_subjects_match': oof_subjects
        == proposal_subjects,
        'oof_subjects_exactly_development_train': oof_subjects
        == development_subjects,
        'no_locked_subject_in_local_oof': not bool(oof_subjects & locked_subjects),
        'no_locked_subject_in_proposal_oof': not bool(
            proposal_subjects & locked_subjects
        ),
        'local_rows_complete': len(pooled_local)
        == 102 * len(LOCAL_THRESHOLDS),
        'proposal_rows_complete': len(pooled_proposal)
        == 102 * len(PROPOSAL_THRESHOLDS),
        'each_oof_subject_has_one_fold': bool((
            pooled_local.groupby('subject_id')['development_fold'].nunique()
            == 1
        ).all()),
    }
    checks_or_raise(oof_checks, 'Pooled OOF accounting failed')

    assignment = development_df.set_index('subject_id')['development_fold']
    observed_assignment = pooled_local.drop_duplicates('subject_id').set_index(
        'subject_id'
    )['development_fold']
    if not np.array_equal(
        assignment.loc[sorted(development_subjects)].to_numpy(dtype=int),
        observed_assignment.loc[sorted(development_subjects)].to_numpy(dtype=int),
    ):
        raise RuntimeError('OOF fold assignments disagree with the frozen manifest.')

    local_summary_df, selected_local = summarize_threshold_grid(
        pooled_local, LOCAL_THRESHOLDS, 'probability'
    )
    proposal_summary_df, selected_proposal = summarize_threshold_grid(
        pooled_proposal, PROPOSAL_THRESHOLDS, 'proposal_probability'
    )
    frozen_local_threshold = float(selected_local['threshold'])
    frozen_proposal_threshold = float(selected_proposal['threshold'])
    selected_local_cases = pooled_local[np.isclose(
        pooled_local['threshold'].to_numpy(dtype=float),
        frozen_local_threshold, rtol=0.0, atol=FLOAT_ATOL,
    )].copy()
    selected_proposal_cases = pooled_proposal[np.isclose(
        pooled_proposal['threshold'].to_numpy(dtype=float),
        frozen_proposal_threshold, rtol=0.0, atol=FLOAT_ATOL,
    )].copy()
    selected_local_cases = selected_local_cases.sort_values(
        ['development_fold', 'subject_id']
    ).reset_index(drop=True)
    selected_proposal_cases = selected_proposal_cases.sort_values(
        ['development_fold', 'subject_id']
    ).reset_index(drop=True)
    pooled_metrics = aggregate_case_metrics(selected_local_cases)
    proposal_metrics = aggregate_case_metrics(selected_proposal_cases)

    proposal_payload = selected_proposal_cases[[
        'subject_id', 'development_fold', 'predicted_voxels',
        'intersection_voxels', 'predicted_volume_ml',
        'lesion_true_positive', 'lesion_false_positive',
        'lesion_false_negative', 'component_true_positive',
        'component_total', 'small_component_true_positive',
        'small_component_total',
    ]].copy()
    proposal_payload = proposal_payload.rename(columns={
        column: 'proposal_' + column
        for column in proposal_payload.columns
        if column not in {'subject_id', 'development_fold'}
    })
    selected_combined = selected_local_cases.merge(
        proposal_payload,
        on=['subject_id', 'development_fold'], how='inner', validate='one_to_one',
    )
    selected_combined = selected_combined.rename(columns={
        'threshold': 'frozen_local_threshold',
    })
    selected_combined.insert(
        selected_combined.columns.get_loc('frozen_local_threshold') + 1,
        'frozen_proposal_threshold', frozen_proposal_threshold,
    )

    oof_subject_audit = selected_local_cases[[
        'subject_id', 'development_fold', 'epoch', 'lesion_volume_bin',
        'hard_metric_eligible', 'target_voxels', 'target_volume_ml',
    ]].copy()
    oof_subject_audit['manifest_split'] = oof_subject_audit['subject_id'].map(
        multimodal_df.set_index('subject_id')['dev_split']
    )
    oof_subject_audit['is_unique_oof_subject'] = ~oof_subject_audit[
        'subject_id'
    ].duplicated(keep=False)
    oof_subject_audit['locked_subject'] = oof_subject_audit[
        'subject_id'
    ].isin(locked_subjects)

    fold_metric_rows = []
    for fold in EXPECTED_FOLDS:
        local_fold = selected_local_cases[
            selected_local_cases['development_fold'] == fold
        ]
        proposal_fold = selected_proposal_cases[
            selected_proposal_cases['development_fold'] == fold
        ]
        local_values = aggregate_case_metrics(local_fold)
        proposal_values = aggregate_case_metrics(proposal_fold)
        gate = fold_gates[fold]
        row = {
            'development_fold': fold,
            'heldout_subjects': len(local_fold),
            'best_epoch': int(gate['best_epoch']),
            'fold_selected_local_threshold': float(gate['best_threshold']),
            'frozen_local_threshold': frozen_local_threshold,
            'fold_selected_proposal_threshold': float(
                gate['best_proposal_threshold']
            ),
            'frozen_proposal_threshold': frozen_proposal_threshold,
            **local_values,
            'proposal_global_dice': proposal_values['positive_global_dice'],
            'proposal_component_recall': proposal_values['component_recall'],
            'proposal_small_component_recall': proposal_values[
                'small_component_recall'
            ],
            'frozen_threshold_composite_score': float(
                0.90 * local_values['selection_score']
                + 0.10 * proposal_values['positive_global_dice']
            ),
        }
        fold_metric_rows.append(row)
    fold_metrics_df = pd.DataFrame(fold_metric_rows)

    variability_metrics = [
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'small_case_mean_dice', 'proposal_global_dice',
        'frozen_threshold_composite_score',
    ]
    variability_rows = []
    for metric in variability_metrics:
        values = fold_metrics_df[metric].to_numpy(dtype=float)
        variability_rows.append({
            'metric': metric,
            'unweighted_fold_mean': float(values.mean()),
            'fold_standard_deviation': float(values.std(ddof=1)),
            'fold_minimum': float(values.min()),
            'fold_maximum': float(values.max()),
            'folds': len(values),
            'primary_estimate_is_pooled_oof_not_fold_mean': True,
        })
    fold_variability_df = pd.DataFrame(variability_rows)
    lesion_bins_df = lesion_bin_metrics(selected_local_cases)
    bootstrap_df = bootstrap_confidence_intervals(selected_local_cases)

    threshold_checks = {
        'local_threshold_is_grid_member': bool(np.isclose(
            LOCAL_THRESHOLDS, frozen_local_threshold,
            rtol=0.0, atol=FLOAT_ATOL,
        ).any()),
        'proposal_threshold_is_grid_member': bool(np.isclose(
            PROPOSAL_THRESHOLDS, frozen_proposal_threshold,
            rtol=0.0, atol=FLOAT_ATOL,
        ).any()),
        'local_threshold_is_unique_rank_one': int((
            local_summary_df['selection_rank'] == 1
        ).sum()) == 1,
        'proposal_threshold_is_unique_rank_one': int((
            proposal_summary_df['selection_rank'] == 1
        ).sum()) == 1,
        'local_selection_exact_frozen_rule': floats_match(
            selected_local['selection_score'],
            0.60 * pooled_metrics['positive_mean_dice']
            + 0.20 * pooled_metrics['positive_global_dice']
            + 0.15 * pooled_metrics['lesion_f1']
            + 0.05 * pooled_metrics['small_case_mean_dice'],
        ),
        'proposal_selection_exact_frozen_rule': floats_match(
            selected_proposal['proposal_selection_score'],
            0.45 * proposal_metrics['positive_global_dice']
            + 0.25 * proposal_metrics['lesion_f1']
            + 0.15 * proposal_metrics['lesion_precision']
            + 0.15 * proposal_metrics['component_recall'],
        ),
        'selected_local_has_102_subjects': len(selected_local_cases) == 102,
        'selected_proposal_has_102_subjects': len(selected_proposal_cases) == 102,
        'bootstrap_replicates_exact': bool((
            bootstrap_df['bootstrap_replicates'] == BOOTSTRAP_REPLICATES
        ).all()),
        'bootstrap_values_finite': bool(np.isfinite(
            bootstrap_df[[
                'point_estimate', 'ci_95_lower', 'ci_95_upper',
                'bootstrap_standard_error',
            ]].to_numpy(dtype=float)
        ).all()),
    }
    checks_or_raise(threshold_checks, 'Pooled threshold-selection audit failed')

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(
        pd.DataFrame(fold_integrity_rows), FOLD_INTEGRITY_AUDIT_PATH
    )
    atomic_write_csv(oof_subject_audit, OOF_SUBJECT_AUDIT_PATH)
    atomic_write_csv(
        pooled_local.sort_values([
            'development_fold', 'subject_id', 'threshold'
        ]),
        POOLED_LOCAL_CASE_PATH,
    )
    atomic_write_csv(
        pooled_proposal.sort_values([
            'development_fold', 'subject_id', 'threshold'
        ]),
        POOLED_PROPOSAL_CASE_PATH,
    )
    atomic_write_csv(local_summary_df, LOCAL_THRESHOLD_SUMMARY_PATH)
    atomic_write_csv(proposal_summary_df, PROPOSAL_THRESHOLD_SUMMARY_PATH)
    atomic_write_csv(selected_combined, SELECTED_OOF_CASE_PATH)
    atomic_write_csv(fold_metrics_df, FOLD_METRICS_PATH)
    atomic_write_csv(fold_variability_df, FOLD_VARIABILITY_PATH)
    atomic_write_csv(lesion_bins_df, LESION_BIN_METRICS_PATH)
    atomic_write_csv(bootstrap_df, BOOTSTRAP_CI_PATH)
    atomic_write_csv(pd.DataFrame(source_records), SOURCE_MANIFEST_PATH)

    deployment_policy = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'aggregation_version': AGGREGATION_VERSION,
        'code_revision': AGGREGATION_CODE_REVISION,
        'training_series_version': TRAINING_SERIES_VERSION,
        'selected_architecture': 'R5 VascuFateNet V2.2 global-to-local arterial',
        'parameter_count_per_model': EXPECTED_PARAMETERS,
        'ensemble_members': 5,
        'ensemble_checkpoints': checkpoint_records,
        'checkpoint_weight_state': 'ema_model_state',
        'ensemble_probability_rule': (
            'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'segmentation_decision_rule': (
            'ensemble_mean_probability_greater_than_or_equal_to_'
            'frozen_local_threshold'
        ),
        'frozen_local_threshold': frozen_local_threshold,
        'frozen_proposal_threshold': frozen_proposal_threshold,
        'proposal_threshold_role': 'diagnostic_only_not_final_segmentation',
        'threshold_source': (
            'pooled_102_subject_development_out_of_fold_sufficient_statistics'
        ),
        'threshold_selection_rule': (
            'maximize_0.60_mean_dice_plus_0.20_global_dice_plus_'
            '0.15_lesion_f1_plus_0.05_tiny_small_subject_mean;_'
            'tie_break_higher_mean_dice_then_lower_threshold'
        ),
        'proposal_threshold_selection_rule': (
            'maximize_0.45_global_dice_plus_0.25_lesion_f1_plus_'
            '0.15_lesion_precision_plus_0.15_component_recall;_'
            'tie_break_higher_component_recall_then_lower_threshold'
        ),
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'locked_evaluation_splits': ['val', 'test'],
        'locked_evaluation_execution': (
            'evaluate_validation_and_test_together_once_with_no_intervening_'
            'model_threshold_or_postprocessing_change'
        ),
        'primary_oof_metrics': {
            'positive_case_mean_dice': pooled_metrics['positive_mean_dice'],
            'positive_case_median_dice': pooled_metrics['positive_median_dice'],
            'pooled_global_dice': pooled_metrics['positive_global_dice'],
            'lesion_f1': pooled_metrics['lesion_f1'],
            'tiny_small_subject_weighted_mean_dice': pooled_metrics[
                'small_case_mean_dice'
            ],
            'component_recall': pooled_metrics['component_recall'],
            'small_component_recall': pooled_metrics[
                'small_component_recall'
            ],
        },
        'reporting_contract': {
            'primary_cross_validation_estimate': 'pooled_subject_level_oof',
            'fold_mean_and_standard_deviation': 'secondary_variability',
            'confidence_intervals': (
                'descriptive_subject_bootstrap_conditional_on_frozen_threshold'
            ),
            'fold0_architecture_development_bias_acknowledged': True,
            'cross_validation_is_not_locked_test_performance': True,
            'confidence_intervals_do_not_account_for_threshold_selection': True,
        },
        'changes_after_freeze_prohibited': [
            'architecture', 'checkpoint', 'ensemble_members', 'weight_state',
            'threshold', 'postprocessing', 'test_time_augmentation',
            'metric_definitions', 'split_membership',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'raw_target_arrays_read': False,
        'precomputed_target_derived_oof_statistics_read': True,
        'training_performed': False,
        'model_constructed': False,
        'optimizer_constructed': False,
        'locked_evaluation_authorized': True,
    }
    atomic_write_json(deployment_policy, DEPLOYMENT_POLICY_PATH)

    output_paths = {
        'fold_integrity_audit': FOLD_INTEGRITY_AUDIT_PATH,
        'oof_subject_audit': OOF_SUBJECT_AUDIT_PATH,
        'pooled_local_case_statistics': POOLED_LOCAL_CASE_PATH,
        'pooled_proposal_case_statistics': POOLED_PROPOSAL_CASE_PATH,
        'local_threshold_summary': LOCAL_THRESHOLD_SUMMARY_PATH,
        'proposal_threshold_summary': PROPOSAL_THRESHOLD_SUMMARY_PATH,
        'selected_oof_case_metrics': SELECTED_OOF_CASE_PATH,
        'fold_metrics': FOLD_METRICS_PATH,
        'fold_variability': FOLD_VARIABILITY_PATH,
        'lesion_bin_metrics': LESION_BIN_METRICS_PATH,
        'bootstrap_confidence_intervals': BOOTSTRAP_CI_PATH,
        'source_artifact_manifest': SOURCE_MANIFEST_PATH,
        'frozen_deployment_policy': DEPLOYMENT_POLICY_PATH,
    }
    output_manifest = {
        name: {'path': str(path), 'sha256': file_sha256(path)}
        for name, path in output_paths.items()
    }

    source_unchanged_checks = source_stats_unchanged(source_records)
    checks_or_raise(
        source_unchanged_checks,
        'A source artifact changed while the read-only aggregation was running',
    )

    final_checks = {
        **upstream_checks,
        **manifest_checks,
        **oof_checks,
        **threshold_checks,
        'all_fold_reconciliations_passed': bool(
            fold_reconciliation_checks
            and all(fold_reconciliation_checks.values())
        ),
        'all_source_stats_unchanged': all(source_unchanged_checks.values()),
        'all_output_artifacts_exist': all(
            path.is_file() for path in output_paths.values()
        ),
        'output_root_is_outside_training_series': not str(
            OUTPUT_ROOT.resolve()
        ).startswith(str(TRAINING_SERIES_ROOT.resolve()) + os.sep),
        'raw_target_arrays_not_read': True,
        'locked_validation_targets_not_read': True,
        'test_targets_not_read': True,
        'model_not_constructed': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
    }
    checks_or_raise(final_checks, 'Final OOF aggregation authorization failed')

    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'aggregation_version': AGGREGATION_VERSION,
        'code_revision': AGGREGATION_CODE_REVISION,
        'training_series_version': TRAINING_SERIES_VERSION,
        'aggregation_complete': True,
        'deployment_threshold_frozen': True,
        'locked_evaluation_authorized': True,
        'development_folds': list(EXPECTED_FOLDS),
        'oof_subjects': 102,
        'frozen_local_threshold': frozen_local_threshold,
        'frozen_proposal_threshold': frozen_proposal_threshold,
        'pooled_oof_metrics': pooled_metrics,
        'pooled_proposal_metrics': proposal_metrics,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'bootstrap_seed': BOOTSTRAP_SEED,
        'bootstrap_interpretation': (
            'descriptive_subject_level_percentile_intervals_conditional_on_'
            'the_frozen_threshold;_does_not_include_model_selection_or_'
            'threshold_selection_uncertainty'
        ),
        'fold0_architecture_development_bias_acknowledged': True,
        'cross_validation_is_not_locked_test_performance': True,
        'source_artifact_manifest': to_builtin(source_records),
        'output_artifact_manifest': output_manifest,
        'final_checks': final_checks,
        'raw_target_arrays_read': False,
        'precomputed_target_derived_oof_statistics_read': True,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'model_constructed': False,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'next_action': (
            'Run the separately isolated locked validation/test five-model '
            'ensemble evaluation exactly once under the frozen deployment policy.'
        ),
    }
    atomic_write_json(gate, AGGREGATION_GATE_PATH)

    print('\nFold integrity summary:')
    display(pd.DataFrame(fold_integrity_rows)[[
        'development_fold', 'fit_subjects', 'heldout_subjects', 'best_epoch',
        'fold_selected_local_threshold', 'fold_best_mean_dice',
        'fold_best_global_dice', 'strict_numeric_best_verified',
    ]])
    print('\nTop pooled local thresholds:')
    display(local_summary_df.sort_values('selection_rank').head(5)[[
        'selection_rank', 'threshold', 'selection_score',
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'small_case_mean_dice', 'component_recall',
    ]])
    print('\nFold metrics at the single frozen threshold:')
    display(fold_metrics_df[[
        'development_fold', 'heldout_subjects', 'frozen_local_threshold',
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'small_case_mean_dice', 'component_recall',
    ]])
    print('\nLesion-volume-bin metrics:')
    display(lesion_bins_df)
    print('\nDescriptive 95% subject-bootstrap intervals:')
    display(bootstrap_df)

    print('\n' + '=' * 80)
    print('TRIAL8 FINAL FIVE-FOLD OOF AGGREGATION AND THRESHOLD FREEZE PASSED')
    print('=' * 80)
    print(f'OOF subjects:                        {len(selected_local_cases)}')
    print(f'Frozen local threshold:              {frozen_local_threshold:.2f}')
    print(f'Frozen proposal threshold:           {frozen_proposal_threshold:.2f}')
    print(
        'Positive-case mean / median Dice:   '
        f'{pooled_metrics["positive_mean_dice"]:.4f} / '
        f'{pooled_metrics["positive_median_dice"]:.4f}'
    )
    print(
        'Pooled/global Dice:                 '
        f'{pooled_metrics["positive_global_dice"]:.4f}'
    )
    print(f'Lesion F1:                           {pooled_metrics["lesion_f1"]:.4f}')
    print(
        'Tiny/small subject mean Dice:       '
        f'{pooled_metrics["small_case_mean_dice"]:.4f}'
    )
    print(
        'All / small component recall:       '
        f'{pooled_metrics["component_recall"]:.4f} / '
        f'{pooled_metrics["small_component_recall"]:.4f}'
    )
    print('Five-model EMA ensemble frozen:       True')
    print('Postprocessing / TTA:                 none / none')
    print('Raw target arrays read:               False')
    print('Locked validation targets read:       False')
    print('Test targets read:                    False')
    print('Model / optimizer constructed:        False / False')
    print('Training performed:                   False')
    print(f'Deployment policy:                   {DEPLOYMENT_POLICY_PATH}')
    print(f'Aggregation gate:                    {AGGREGATION_GATE_PATH}')
    print(
        '\nNext: run the isolated locked validation/test ensemble evaluation '
        'exactly once. Do not tune after viewing either locked split.'
    )


run_oof_aggregation()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )















LOCKED_EVALUATION_VERSION = 'v2_final_locked_val_test_fivefold_ensemble_once'
LOCKED_EVALUATION_CODE_REVISION = (
    'r2_locked_input_contract_training_lock_semantics'
)
OOF_FREEZE_VERSION = 'v1_final_fivefold_oof_threshold_freeze'
EXPECTED_OOF_AGGREGATION_VERSION = OOF_FREEZE_VERSION
EXPECTED_OOF_CODE_REVISION = (
    'r1_exact_sufficient_statistics_locked_policy'
)
EXPECTED_ENSEMBLE_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_LOCKED_SPLITS = {'val': 23, 'test': 23}
EXPECTED_FROZEN_LOCAL_THRESHOLD = 0.40
EXPECTED_FROZEN_PROPOSAL_THRESHOLD = 0.35
LOCKED_BOOTSTRAP_REPLICATES = 10_000
LOCKED_BOOTSTRAP_SEED = 20261012

OOF_FREEZE_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / OOF_FREEZE_VERSION
)
OOF_DEPLOYMENT_POLICY_PATH = OOF_FREEZE_ROOT / 'frozen_deployment_policy.json'
OOF_AGGREGATION_GATE_PATH = OOF_FREEZE_ROOT / 'oof_threshold_freeze_gate.json'

LOCKED_OUTPUT_ROOT = (
    PROJECT_ROOT / 'evaluation' / MODEL_VERSION / LOCKED_EVALUATION_VERSION
)
LOCKED_PREDICTION_ROOT = LOCKED_OUTPUT_ROOT / 'target_blind_predictions'
LOCKED_PREDICTION_INDEX_PATH = LOCKED_OUTPUT_ROOT / 'prediction_index.json'
LOCKED_PREDICTION_GATE_PATH = (
    LOCKED_OUTPUT_ROOT / 'target_blind_prediction_gate.json'
)
LOCKED_SOURCE_MANIFEST_PATH = LOCKED_OUTPUT_ROOT / 'source_artifact_manifest.csv'
LOCKED_PREDICTION_AUDIT_PATH = LOCKED_OUTPUT_ROOT / 'prediction_audit.csv'
LOCKED_CASE_METRICS_PATH = LOCKED_OUTPUT_ROOT / 'locked_case_metrics.csv'
LOCKED_COMPONENT_METRICS_PATH = (
    LOCKED_OUTPUT_ROOT / 'locked_component_metrics.csv'
)
LOCKED_SPLIT_SUMMARY_PATH = LOCKED_OUTPUT_ROOT / 'locked_split_summary.csv'
LOCKED_BIN_METRICS_PATH = LOCKED_OUTPUT_ROOT / 'locked_lesion_bin_metrics.csv'
LOCKED_MEMBER_METRICS_PATH = (
    LOCKED_OUTPUT_ROOT / 'ensemble_member_metrics.csv'
)
LOCKED_DIVERSITY_PATH = LOCKED_OUTPUT_ROOT / 'ensemble_diversity_metrics.csv'
LOCKED_BOOTSTRAP_PATH = LOCKED_OUTPUT_ROOT / 'locked_bootstrap_intervals.csv'
LOCKED_GENERALIZATION_PATH = (
    LOCKED_OUTPUT_ROOT / 'oof_to_locked_generalization_summary.csv'
)
LOCKED_REPORT_PATH = LOCKED_OUTPUT_ROOT / 'locked_evaluation_report.json'
LOCKED_FINAL_GATE_PATH = LOCKED_OUTPUT_ROOT / 'locked_evaluation_gate.json'


def locked_checks_or_raise(checks, context):
    failed = [name for name, passed in checks.items() if not bool(passed)]
    if failed:
        raise RuntimeError(context + ': ' + ', '.join(failed))


def locked_canonical_hash(record):
    payload = json.dumps(
        to_builtin(record), sort_keys=True, separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def locked_source_record(path, role, development_fold=None):
    path = Path(path)
    stat = path.stat()
    return {
        'role': str(role),
        'development_fold': (
            int(development_fold) if development_fold is not None else -1
        ),
        'path': str(path),
        'sha256': file_sha256(path),
        'size_bytes': int(stat.st_size),
        'mtime_ns': int(stat.st_mtime_ns),
    }


def locked_sources_unchanged(records):
    checks = {}
    for index, record in enumerate(records):
        path = Path(record['path'])
        stat = path.stat() if path.exists() else None
        checks[f'source_{index:03d}_{record["role"]}_unchanged'] = bool(
            stat is not None
            and int(stat.st_size) == int(record['size_bytes'])
            and int(stat.st_mtime_ns) == int(record['mtime_ns'])
            and file_sha256(path) == record['sha256']
        )
    return checks


def locked_manifest_input_contract_checks(locked_rows):
    
    required_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing = sorted(required_columns - set(locked_rows.columns))
    if missing:
        raise RuntimeError(
            'Locked manifest is missing required inference columns: '
            + ', '.join(missing)
        )

    path_columns = [
        'vascufate_v2_npz_path', 'trial8_graph_sidecar_path',
        'trial8_clinical_sidecar_path',
    ]
    paths_declared = locked_rows[path_columns].apply(
        lambda column: column.map(
            lambda value: isinstance(value, str) and bool(value.strip())
        )
    )
    paths_exist = locked_rows[path_columns].apply(
        lambda column: column.map(
            lambda value: isinstance(value, str) and Path(value).is_file()
        )
    )
    hash_columns = [
        'trial8_npz_sha256', 'trial8_graph_sidecar_sha256',
        'trial8_clinical_sidecar_sha256',
    ]
    hashes_declared = locked_rows[hash_columns].apply(
        lambda column: column.map(
            lambda value: (
                isinstance(value, str)
                and len(value.strip()) == 64
                and all(character in '0123456789abcdef'
                        for character in value.strip().lower())
            )
        )
    )
    atlas_paths_valid = []
    for row in locked_rows.to_dict('records'):
        if as_bool(row['trial8_atlas_enabled']):
            atlas_path = str(row['trial8_atlas_vascular30_path']).strip()
            atlas_paths_valid.append(bool(atlas_path) and Path(atlas_path).is_file())
        else:
            atlas_paths_valid.append(True)

    loader_passed = bool(
        locked_rows['trial8_v2_multimodal_loader_preflight_passed']
        .map(as_bool).all()
    )
    training_locked = not bool(
        locked_rows['trial8_model_training_authorized'].map(as_bool).any()
    )
    inputs_verified = bool(
        loader_passed
        and training_locked
        and paths_declared.to_numpy(dtype=bool).all()
        and paths_exist.to_numpy(dtype=bool).all()
        and hashes_declared.to_numpy(dtype=bool).all()
        and all(atlas_paths_valid)
    )
    return {
        'all_loader_preflights_passed': loader_passed,
        'training_remains_locked_for_all': training_locked,
        'all_model_input_paths_declared': bool(
            paths_declared.to_numpy(dtype=bool).all()
        ),
        'all_model_input_files_exist': bool(
            paths_exist.to_numpy(dtype=bool).all()
        ),
        'all_model_input_hashes_declared': bool(
            hashes_declared.to_numpy(dtype=bool).all()
        ),
        'all_enabled_atlas_paths_exist': bool(all(atlas_paths_valid)),
        'all_target_free_model_inputs_verified': inputs_verified,
    }


def locked_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return np.packbits(mask.reshape(-1), bitorder='little')


def locked_unpack_mask(bits, shape):
    count = int(np.prod(shape))
    values = np.unpackbits(
        np.asarray(bits, dtype=np.uint8), bitorder='little'
    )[:count]
    return values.reshape(tuple(int(v) for v in shape)).astype(bool)


def locked_atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def locked_prediction_path(subject_id):
    return LOCKED_PREDICTION_ROOT / f'{subject_id}_fivefold_ensemble_masks.npz'


def locked_load_target_free_case(row):
    
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    native_shape = tuple(int(v) for v in image.shape[1:])
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(native_shape, dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: inference padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id,
        'row': row,
        'image_canvas': image_canvas,
        'region_canvas': region_canvas,
        'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas,
        'lvo_present': lvo_present,
        'native_shape': native_shape,
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


@torch.no_grad()
def locked_infer_case(model, row, device):
    case = locked_load_target_free_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(model, base_batch)
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][axis] + native_start[axis])
                    for axis in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(model, batch, cache)
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(
                        native_start[axis],
                        min(
                            native_start[axis] + LOCAL_PATCH_SIZE[axis],
                            native_shape[axis],
                        ),
                    )
                    for axis in range(3)
                )
                copied_shape = tuple(
                    item.stop - item.start for item in destination
                )
                source = tuple(
                    slice(0, copied_shape[axis]) for axis in range(3)
                )
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete sliding-window coverage."
        )
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(
            case['pad_start'][axis],
            case['pad_start'][axis] + native_shape[axis],
        )
        for axis in range(3)
    )
    proposal = proposal_canvas[native_slices].astype(np.float32)
    presence = float(torch.sigmoid(
        cache['routed']['lesion_presence_logit'].float()
    )[0].cpu())
    routed_volume_ml = float(torch.expm1(
        cache['routed']['log_volume_prediction'].float()
    )[0].clamp_min(0).cpu())
    del base_batch, cache
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float32),
        'proposal_probability': proposal,
        'presence_probability': presence,
        'routed_volume_ml': routed_volume_ml,
    }


def locked_validate_prediction_archive(path, expected_contract_hash, subject_id):
    path = Path(path)
    if not path.is_file():
        return None
    with np.load(path, allow_pickle=False) as data:
        required = {
            'subject_id', 'dev_split', 'native_shape', 'contract_sha256',
            'fold_ids', 'fold_local_masks_packed',
            'ensemble_local_mask_packed', 'ensemble_proposal_mask_packed',
            'fold_predicted_voxels', 'fold_presence_probabilities',
            'fold_routed_volume_ml', 'ensemble_local_probability_sha256',
            'ensemble_proposal_probability_sha256',
        }
        if not required.issubset(data.files):
            raise RuntimeError(f'{subject_id}: prediction archive is incomplete.')
        observed_subject = scalar_string(data['subject_id'])
        observed_contract = scalar_string(data['contract_sha256'])
        shape = tuple(int(v) for v in np.asarray(data['native_shape']).tolist())
        fold_ids = tuple(int(v) for v in np.asarray(data['fold_ids']).tolist())
        fold_bits = np.asarray(data['fold_local_masks_packed'], dtype=np.uint8)
        local_bits = np.asarray(
            data['ensemble_local_mask_packed'], dtype=np.uint8
        )
        proposal_bits = np.asarray(
            data['ensemble_proposal_mask_packed'], dtype=np.uint8
        )
    expected_bytes = (int(np.prod(shape)) + 7) // 8
    checks = {
        'subject_matches': observed_subject == subject_id,
        'contract_matches': observed_contract == expected_contract_hash,
        'shape_is_three_dimensional': len(shape) == 3,
        'fold_ids_exact': fold_ids == EXPECTED_ENSEMBLE_FOLDS,
        'fold_mask_shape_exact': fold_bits.shape == (5, expected_bytes),
        'ensemble_local_mask_size_exact': local_bits.size == expected_bytes,
        'ensemble_proposal_mask_size_exact': proposal_bits.size == expected_bytes,
    }
    locked_checks_or_raise(
        checks, f'{subject_id}: existing target-blind prediction rejected'
    )
    return {
        'subject_id': subject_id,
        'path': str(path),
        'sha256': file_sha256(path),
        'native_shape': list(shape),
    }


def locked_pairwise_dice(masks):
    values = []
    for left in range(len(masks)):
        for right in range(left + 1, len(masks)):
            denominator = int(masks[left].sum() + masks[right].sum())
            intersection = int((masks[left] & masks[right]).sum())
            values.append(
                2.0 * intersection / denominator if denominator else 1.0
            )
    return values


def locked_prediction_manifest_row(
    subject_id, split, archive_path, shape, fold_masks,
    ensemble_local, ensemble_proposal, local_probability,
    proposal_probability, fold_presence, fold_routed_volume,
):
    pairwise = locked_pairwise_dice(fold_masks)
    vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
    union_voxels = int((vote_count > 0).sum())
    disagreement_voxels = int(((vote_count > 0) & (vote_count < 5)).sum())
    return {
        'subject_id': subject_id,
        'dev_split': split,
        'prediction_path': str(archive_path),
        'prediction_sha256': file_sha256(archive_path),
        'native_shape': list(shape),
        'ensemble_local_voxels': int(ensemble_local.sum()),
        'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
        'fold_local_predicted_voxels': [int(mask.sum()) for mask in fold_masks],
        'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
        'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
        'disagreement_union_fraction': float(
            disagreement_voxels / max(union_voxels, 1)
        ),
        'ensemble_local_probability_mean': float(local_probability.mean()),
        'ensemble_local_probability_maximum': float(local_probability.max()),
        'ensemble_proposal_probability_mean': float(
            proposal_probability.mean()
        ),
        'ensemble_proposal_probability_maximum': float(
            proposal_probability.max()
        ),
        'mean_presence_probability': float(np.mean(fold_presence)),
        'mean_routed_volume_ml': float(np.mean(fold_routed_volume)),
    }


def locked_load_target(row, expected_shape):
    subject_id = str(row['subject_id'])
    path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(path, allow_pickle=False) as data:
        if not {'target_binary', 'target_soft', 'target_spacing'}.issubset(
            data.files
        ):
            raise RuntimeError(f'{subject_id}: locked target arrays unavailable.')
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    checks = {
        'target_shape_matches_prediction': tuple(target.shape) == tuple(expected_shape),
        'soft_shape_matches_prediction': tuple(soft.shape) == tuple(expected_shape),
        'spacing_has_three_values': spacing.shape == (3,),
        'spacing_is_finite_positive': bool(
            np.isfinite(spacing).all() and (spacing > 0).all()
        ),
        'soft_is_finite': bool(np.isfinite(soft).all()),
    }
    locked_checks_or_raise(checks, f'{subject_id}: locked target integrity failed')
    return target, soft, spacing


def locked_mask_metrics(prediction, target, spacing, probability_role='local'):
    intersection = int((prediction & target).sum())
    predicted = int(prediction.sum())
    target_voxels = int(target.sum())
    denominator = predicted + target_voxels
    dice = 2.0 * intersection / denominator if denominator else 1.0
    lesion_tp, lesion_fp, lesion_fn = pilot_lesion_counts(prediction, target)
    iou_threshold = 0.05 if probability_role == 'proposal' else 0.10
    components = pilot_target_component_iou_records(
        prediction, target, spacing
    ) if target_voxels > 0 else []
    component_tp = sum(
        float(item['maximum_component_iou']) >= iou_threshold
        for item in components
    )
    small_components = [
        item for item in components
        if item['component_tier'] in {
            'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
        }
    ]
    small_tp = sum(
        float(item['maximum_component_iou']) >= iou_threshold
        for item in small_components
    )
    return {
        'intersection_voxels': intersection,
        'predicted_voxels': predicted,
        'target_voxels': target_voxels,
        'dice': float(dice),
        'voxel_precision': intersection / max(predicted, 1),
        'voxel_recall': intersection / max(target_voxels, 1),
        'lesion_true_positive': int(lesion_tp),
        'lesion_false_positive': int(lesion_fp),
        'lesion_false_negative': int(lesion_fn),
        'component_true_positive': int(component_tp),
        'component_total': int(len(components)),
        'small_component_true_positive': int(small_tp),
        'small_component_total': int(len(small_components)),
        'components': components,
    }


def locked_aggregate_case_rows(frame):
    eligible = frame[frame['hard_metric_eligible'].astype(bool)].copy()
    positive = eligible[eligible['target_voxels'].astype(int) > 0].copy()
    tiny_small = positive[positive['lesion_volume_bin'].isin(['tiny', 'small'])]
    intersection = int(positive['intersection_voxels'].sum())
    predicted = int(positive['predicted_voxels'].sum())
    target = int(positive['target_voxels'].sum())
    lesion_tp = int(eligible['lesion_true_positive'].sum())
    lesion_fp = int(eligible['lesion_false_positive'].sum())
    lesion_fn = int(eligible['lesion_false_negative'].sum())
    precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * precision * recall / max(precision + recall, 1e-12)
    )
    component_tp = int(positive['component_true_positive'].sum())
    component_total = int(positive['component_total'].sum())
    small_tp = int(positive['small_component_true_positive'].sum())
    small_total = int(positive['small_component_total'].sum())
    target_volume = float(positive['target_volume_ml'].sum())
    predicted_volume = float(positive['predicted_volume_ml'].sum())
    return {
        'subjects': int(len(frame)),
        'hard_metric_eligible_subjects': int(len(eligible)),
        'positive_subjects': int(len(positive)),
        'positive_mean_dice': float(positive['dice'].mean()) if len(positive) else 0.0,
        'positive_median_dice': float(positive['dice'].median()) if len(positive) else 0.0,
        'positive_global_dice': float(
            2.0 * intersection / max(predicted + target, 1)
        ),
        'lesion_precision': float(precision),
        'lesion_recall': float(recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': component_tp / max(component_total, 1),
        'small_component_recall': small_tp / max(small_total, 1),
        'tiny_small_subject_mean_dice': float(
            tiny_small['dice'].mean()
        ) if len(tiny_small) else 0.0,
        'predicted_to_target_volume_ratio': (
            predicted_volume / max(target_volume, 1e-12)
        ),
        'empty_case_mean_false_positive_volume_ml': float(
            eligible.loc[
                eligible['target_voxels'].astype(int) == 0,
                'predicted_volume_ml',
            ].mean()
        ) if bool((eligible['target_voxels'].astype(int) == 0).any()) else 0.0,
        'intersection_voxels': intersection,
        'predicted_voxels_positive_cases': predicted,
        'target_voxels_positive_cases': target,
        'lesion_true_positive': lesion_tp,
        'lesion_false_positive': lesion_fp,
        'lesion_false_negative': lesion_fn,
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


def locked_bootstrap_split(frame, split, replicates, seed):
    frame = frame.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    metric_names = [
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]
    observed = locked_aggregate_case_rows(frame)
    samples = {name: np.empty(replicates, dtype=np.float64) for name in metric_names}
    for replicate in range(replicates):
        indices = rng.integers(0, len(frame), size=len(frame))
        summary = locked_aggregate_case_rows(frame.iloc[indices])
        for name in metric_names:
            samples[name][replicate] = float(summary[name])
    rows = []
    for name in metric_names:
        values = samples[name]
        rows.append({
            'dev_split': split,
            'metric': name,
            'point_estimate': float(observed[name]),
            'ci_95_lower': float(np.quantile(values, 0.025)),
            'ci_95_upper': float(np.quantile(values, 0.975)),
            'bootstrap_standard_error': float(values.std(ddof=1)),
            'bootstrap_replicates': int(replicates),
            'bootstrap_seed': int(seed),
            'method': (
                'subject_level_nonparametric_percentile_fixed_frozen_policy'
            ),
        })
    return rows


def locked_existing_final_is_valid():
    if not LOCKED_FINAL_GATE_PATH.exists():
        return None
    gate = load_json(LOCKED_FINAL_GATE_PATH)
    expected_outputs = {
        'target_blind_prediction_gate', 'source_artifact_manifest',
        'prediction_audit', 'case_metrics', 'component_metrics',
        'split_summary', 'lesion_bin_metrics', 'ensemble_member_metrics',
        'ensemble_diversity_metrics', 'bootstrap_intervals',
        'oof_to_locked_generalization', 'evaluation_report',
    }
    output_manifest = gate.get('output_artifact_manifest', {})
    source_manifest = gate.get('source_artifact_manifest', [])
    checks = {
        'evaluation_complete': gate.get('evaluation_complete') is True,
        'version_matches': gate.get('evaluation_version') == LOCKED_EVALUATION_VERSION,
        'revision_matches': gate.get('code_revision') == LOCKED_EVALUATION_CODE_REVISION,
        'local_threshold_matches': math.isclose(
            float(gate.get('frozen_local_threshold', float('nan'))),
            EXPECTED_FROZEN_LOCAL_THRESHOLD, rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_matches': math.isclose(
            float(gate.get('frozen_proposal_threshold', float('nan'))),
            EXPECTED_FROZEN_PROPOSAL_THRESHOLD, rel_tol=0.0, abs_tol=1e-12,
        ),
        'validation_and_test_evaluated_together': (
            gate.get('evaluated_splits') == ['val', 'test']
        ),
        'no_tuning': gate.get('post_freeze_tuning_performed') is False,
        'output_manifest_schema_exact': set(output_manifest) == expected_outputs,
        'source_manifest_complete': len(source_manifest) == 21,
        'recorded_final_checks_all_true': bool(
            gate.get('final_checks')
            and all(gate.get('final_checks', {}).values())
        ),
        'artifacts_stable': all(
            Path(record['path']).is_file()
            and file_sha256(Path(record['path'])) == record['sha256']
            for record in output_manifest.values()
        ),
        'sources_stable': all(
            Path(record['path']).is_file()
            and file_sha256(Path(record['path'])) == record['sha256']
            for record in source_manifest
        ),
    }
    locked_checks_or_raise(
        checks,
        'Existing locked-evaluation gate failed integrity verification; '
        'refusing to rerun or overwrite',
    )
    return gate


def run_locked_fivefold_evaluation():
    existing = locked_existing_final_is_valid()
    if existing is not None:
        summaries = existing['split_summaries']
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL LOCKED VALIDATION/TEST EVALUATION ALREADY COMPLETE')
        print('=' * 80)
        for split in ['val', 'test']:
            item = summaries[split]
            print(
                f'{split.upper():10s} mean/global Dice:             '
                f'{item["positive_mean_dice"]:.4f} / '
                f'{item["positive_global_dice"]:.4f}'
            )
        print(f'Final gate: {LOCKED_FINAL_GATE_PATH}')
        print('No checkpoint, prediction, target, or result artifact was rewritten.')
        return

    required_paths = [
        OOF_DEPLOYMENT_POLICY_PATH, OOF_AGGREGATION_GATE_PATH,
        MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
        MULTIMODAL_GATE_PATH,
    ]
    missing = [str(path) for path in required_paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(
            'Locked-evaluation prerequisites are missing:\n' + '\n'.join(missing)
        )
    policy = load_json(OOF_DEPLOYMENT_POLICY_PATH)
    oof_gate = load_json(OOF_AGGREGATION_GATE_PATH)
    policy_hash = file_sha256(OOF_DEPLOYMENT_POLICY_PATH)
    source_records = [
        locked_source_record(OOF_DEPLOYMENT_POLICY_PATH, 'deployment_policy'),
        locked_source_record(OOF_AGGREGATION_GATE_PATH, 'oof_freeze_gate'),
        locked_source_record(MULTIMODAL_MANIFEST_PATH, 'multimodal_manifest'),
        locked_source_record(FOLD_MANIFEST_PATH, 'fold_manifest'),
        locked_source_record(GRAPH_SCHEMA_PATH, 'graph_schema'),
        locked_source_record(MULTIMODAL_GATE_PATH, 'multimodal_gate'),
    ]
    checkpoints = sorted(
        policy.get('ensemble_checkpoints', []),
        key=lambda item: int(item['development_fold']),
    )
    for record in checkpoints:
        source_records.append(locked_source_record(
            record['best_checkpoint_path'], 'best_ema_checkpoint',
            record['development_fold'],
        ))
        graph_scaler = Path(resolve_gate_mapping(
            multimodal_gate['graph_scaler_paths'], record['development_fold']
        ))
        clinical_scaler = Path(resolve_gate_mapping(
            multimodal_gate['clinical_scaler_paths'], record['development_fold']
        ))
        source_records.append(locked_source_record(
            graph_scaler, 'fold_graph_scaler', record['development_fold']
        ))
        source_records.append(locked_source_record(
            clinical_scaler, 'fold_clinical_scaler', record['development_fold']
        ))
    local_threshold = float(policy.get('frozen_local_threshold', float('nan')))
    proposal_threshold = float(
        policy.get('frozen_proposal_threshold', float('nan'))
    )
    checkpoint_folds = tuple(
        int(record['development_fold']) for record in checkpoints
    )
    upstream_checks = {
        'oof_aggregation_complete': oof_gate.get('aggregation_complete') is True,
        'locked_evaluation_authorized': (
            oof_gate.get('locked_evaluation_authorized') is True
            and policy.get('locked_evaluation_authorized') is True
        ),
        'aggregation_version_matches': (
            oof_gate.get('aggregation_version') == EXPECTED_OOF_AGGREGATION_VERSION
            and policy.get('aggregation_version') == EXPECTED_OOF_AGGREGATION_VERSION
        ),
        'aggregation_code_revision_matches': (
            oof_gate.get('code_revision') == EXPECTED_OOF_CODE_REVISION
            and policy.get('code_revision') == EXPECTED_OOF_CODE_REVISION
        ),
        'policy_hash_matches_oof_output_manifest': (
            oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('path') == str(OOF_DEPLOYMENT_POLICY_PATH)
            and oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('sha256') == policy_hash
        ),
        'oof_final_checks_all_true': bool(
            oof_gate.get('final_checks')
            and all(oof_gate.get('final_checks', {}).values())
        ),
        'five_checkpoints_exact': checkpoint_folds == EXPECTED_ENSEMBLE_FOLDS,
        'ema_weight_state_exact': all(
            record.get('weight_state') == 'ema_model_state'
            for record in checkpoints
        ) and policy.get('checkpoint_weight_state') == 'ema_model_state',
        'checkpoint_hashes_match_policy': all(
            file_sha256(record['best_checkpoint_path'])
            == record['best_checkpoint_sha256']
            for record in checkpoints
        ),
        'checkpoint_parameters_match': all(
            int(record['parameter_count']) == EXPECTED_V22_PARAMETERS
            for record in checkpoints
        ),
        'local_threshold_exact': math.isclose(
            local_threshold, EXPECTED_FROZEN_LOCAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_exact': math.isclose(
            proposal_threshold, EXPECTED_FROZEN_PROPOSAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'probability_ensemble_rule_exact': (
            policy.get('ensemble_probability_rule')
            == 'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing_none': policy.get('postprocessing') == 'none',
        'tta_none': policy.get('test_time_augmentation') == 'none',
        'both_locked_splits_required': (
            policy.get('locked_evaluation_splits') == ['val', 'test']
        ),
        'sources_initially_stable': all(
            locked_sources_unchanged(source_records).values()
        ),
    }
    locked_checks_or_raise(
        upstream_checks, 'Frozen deployment-policy authorization failed'
    )

    manifest = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    manifest['subject_id'] = manifest['subject_id'].astype(str)
    locked_rows = manifest[
        manifest['dev_split'].astype(str).isin(['val', 'test'])
    ].copy().sort_values(['dev_split', 'subject_id'])
    split_counts = locked_rows['dev_split'].value_counts().to_dict()
    manifest_checks = {
        'locked_subject_count_is_46': len(locked_rows) == 46,
        'locked_subjects_unique': locked_rows['subject_id'].is_unique,
        'split_counts_exact': split_counts == EXPECTED_LOCKED_SPLITS,
        **locked_manifest_input_contract_checks(locked_rows),
    }
    locked_checks_or_raise(manifest_checks, 'Locked manifest audit failed')

    prediction_contract = {
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'deployment_policy_sha256': policy_hash,
        'checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_folds': list(checkpoint_folds),
        'weight_state': 'ema_model_state',
        'ensemble_rule': 'arithmetic_mean_of_five_fold_ema_probabilities',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'subject_ids': locked_rows['subject_id'].tolist(),
        'target_access_before_prediction_commit': False,
    }
    prediction_contract_hash = locked_canonical_hash(prediction_contract)
    LOCKED_PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)
    LOCKED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    if LOCKED_PREDICTION_GATE_PATH.exists():
        prediction_gate = load_json(LOCKED_PREDICTION_GATE_PATH)
        gate_checks = {
            'prediction_complete': prediction_gate.get('prediction_complete') is True,
            'contract_matches': (
                prediction_gate.get('prediction_contract_sha256')
                == prediction_contract_hash
            ),
            'all_46_subjects': prediction_gate.get('subjects_predicted') == 46,
            'locked_targets_not_read': (
                prediction_gate.get('locked_validation_targets_read') is False
                and prediction_gate.get('test_targets_read') is False
            ),
            'prediction_archives_stable': all(
                Path(record['path']).is_file()
                and file_sha256(record['path']) == record['sha256']
                for record in prediction_gate.get('prediction_manifest', [])
            ),
        }
        locked_checks_or_raise(
            gate_checks, 'Existing target-blind prediction gate rejected'
        )
        prediction_manifest = prediction_gate['prediction_manifest']
        print('\nVerified complete target-blind prediction gate; skipping inference.')
    else:
        if not torch.cuda.is_available():
            raise RuntimeError('Final five-model evaluation requires a CUDA GPU.')
        device = torch.device('cuda')
        set_determinism(PILOT_SEED + 9_000)
        gc.collect()
        torch.cuda.empty_cache()
        models = []
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL TARGET-BLIND FIVE-MODEL INFERENCE STARTING')
        print('=' * 80)
        print('Locked targets read:                False')
        print('Optimizer constructed / steps:      False / 0')
        for record in checkpoints:
            fold = int(record['development_fold'])
            checkpoint = torch.load(
                record['best_checkpoint_path'], map_location='cpu',
                weights_only=False,
            )
            checkpoint_checks = {
                'fold_matches': checkpoint.get('development_fold') == fold,
                'best_epoch_matches': int(checkpoint.get('epoch', -1))
                == int(record['best_epoch']),
                'parameter_count_matches': int(
                    checkpoint.get('parameter_count', -1)
                ) == EXPECTED_V22_PARAMETERS,
                'training_series_matches': (
                    checkpoint.get('replication_series_version')
                    == REPLICATION_SERIES_VERSION
                ),
                'training_revision_matches': (
                    checkpoint.get('code_revision') == PILOT_CODE_REVISION
                ),
                'checkpoint_policy_is_strict_numeric': (
                    checkpoint.get('checkpoint_selection_policy')
                    == 'strict_numerical_maximum'
                ),
                'ema_state_present': isinstance(
                    checkpoint.get('ema_model_state'), dict
                ),
            }
            locked_checks_or_raise(
                checkpoint_checks, f'Fold-{fold} checkpoint identity failed'
            )
            model = VascuFateNetV22(pilot_schema)
            if sum(parameter.numel() for parameter in model.parameters()) != (
                EXPECTED_V22_PARAMETERS
            ):
                raise RuntimeError(f'Fold-{fold}: model parameter count changed.')
            model.load_state_dict(checkpoint['ema_model_state'], strict=True)
            model = model.to(device).eval()
            for parameter in model.parameters():
                parameter.requires_grad_(False)
            if not all(
                bool(torch.isfinite(parameter).all().item())
                for parameter in model.parameters()
            ):
                raise RuntimeError(f'Fold-{fold}: EMA parameters are non-finite.')
            models.append((fold, model))
            del checkpoint
            print(f'Loaded fold-{fold} strict-best EMA checkpoint.')

        prior_index = (
            load_json(LOCKED_PREDICTION_INDEX_PATH)
            if LOCKED_PREDICTION_INDEX_PATH.exists() else {}
        )
        if prior_index and prior_index.get('prediction_contract_sha256') != (
            prediction_contract_hash
        ):
            raise RuntimeError(
                'Partial prediction index belongs to another contract; '
                'refusing to overwrite it.'
            )
        prediction_manifest = []
        for index, row in enumerate(locked_rows.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            split = str(row['dev_split'])
            archive_path = locked_prediction_path(subject_id)
            existing_archive = locked_validate_prediction_archive(
                archive_path, prediction_contract_hash, subject_id
            )
            if existing_archive is not None:
                prediction_manifest.append(existing_archive)
                print(
                    f'Target-blind prediction {index:02d}/46: {subject_id} '
                    '(verified resume)'
                )
                continue
            local_sum = None
            proposal_sum = None
            fold_masks = []
            fold_presence = []
            fold_routed_volume = []
            for fold, model in models:
                global RUNTIME_FOLD
                RUNTIME_FOLD = fold
                result = locked_infer_case(model, row, device)
                local_probability = result['probability']
                proposal_probability = result['proposal_probability']
                if local_sum is None:
                    local_sum = np.zeros_like(local_probability, dtype=np.float32)
                    proposal_sum = np.zeros_like(
                        proposal_probability, dtype=np.float32
                    )
                local_sum += local_probability
                proposal_sum += proposal_probability
                fold_masks.append(local_probability >= local_threshold)
                fold_presence.append(result['presence_probability'])
                fold_routed_volume.append(result['routed_volume_ml'])
                del result, local_probability, proposal_probability
            ensemble_local_probability = local_sum / 5.0
            ensemble_proposal_probability = proposal_sum / 5.0
            ensemble_local = ensemble_local_probability >= local_threshold
            ensemble_proposal = (
                ensemble_proposal_probability >= proposal_threshold
            )
            shape = tuple(int(v) for v in ensemble_local.shape)
            fold_packed = np.stack([
                locked_pack_mask(mask) for mask in fold_masks
            ], axis=0)
            local_probability_hash = hashlib.sha256(
                np.ascontiguousarray(
                    ensemble_local_probability.astype('<f4')
                ).tobytes()
            ).hexdigest()
            proposal_probability_hash = hashlib.sha256(
                np.ascontiguousarray(
                    ensemble_proposal_probability.astype('<f4')
                ).tobytes()
            ).hexdigest()
            locked_atomic_npz(
                archive_path,
                subject_id=np.asarray(subject_id, dtype=np.str_),
                dev_split=np.asarray(split, dtype=np.str_),
                native_shape=np.asarray(shape, dtype=np.int64),
                contract_sha256=np.asarray(
                    prediction_contract_hash, dtype=np.str_
                ),
                fold_ids=np.asarray(EXPECTED_ENSEMBLE_FOLDS, dtype=np.int64),
                fold_local_masks_packed=fold_packed,
                ensemble_local_mask_packed=locked_pack_mask(ensemble_local),
                ensemble_proposal_mask_packed=locked_pack_mask(
                    ensemble_proposal
                ),
                fold_predicted_voxels=np.asarray([
                    int(mask.sum()) for mask in fold_masks
                ], dtype=np.int64),
                fold_presence_probabilities=np.asarray(
                    fold_presence, dtype=np.float32
                ),
                fold_routed_volume_ml=np.asarray(
                    fold_routed_volume, dtype=np.float32
                ),
                ensemble_local_probability_sha256=np.asarray(
                    local_probability_hash, dtype=np.str_
                ),
                ensemble_proposal_probability_sha256=np.asarray(
                    proposal_probability_hash, dtype=np.str_
                ),
            )
            manifest_row = locked_prediction_manifest_row(
                subject_id, split, archive_path, shape, fold_masks,
                ensemble_local, ensemble_proposal,
                ensemble_local_probability, ensemble_proposal_probability,
                fold_presence, fold_routed_volume,
            )
            prediction_manifest.append(manifest_row)
            atomic_write_json({
                'prediction_contract_sha256': prediction_contract_hash,
                'completed_subjects': len(prediction_manifest),
                'prediction_manifest': prediction_manifest,
                'locked_validation_targets_read': False,
                'test_targets_read': False,
            }, LOCKED_PREDICTION_INDEX_PATH)
            print(
                f'Target-blind prediction {index:02d}/46: {subject_id} '
                f'({split})'
            )
            del (
                local_sum, proposal_sum, fold_masks,
                ensemble_local_probability, ensemble_proposal_probability,
                ensemble_local, ensemble_proposal,
            )
            gc.collect()
            torch.cuda.empty_cache()

        
        normalized_manifest = []
        for row in locked_rows.to_dict('records'):
            subject_id = str(row['subject_id'])
            archive_path = locked_prediction_path(subject_id)
            locked_validate_prediction_archive(
                archive_path, prediction_contract_hash, subject_id
            )
            with np.load(archive_path, allow_pickle=False) as data:
                shape = tuple(int(v) for v in data['native_shape'].tolist())
                fold_masks = [
                    locked_unpack_mask(bits, shape)
                    for bits in data['fold_local_masks_packed']
                ]
                ensemble_local = locked_unpack_mask(
                    data['ensemble_local_mask_packed'], shape
                )
                ensemble_proposal = locked_unpack_mask(
                    data['ensemble_proposal_mask_packed'], shape
                )
                fold_presence = np.asarray(
                    data['fold_presence_probabilities'], dtype=np.float32
                )
                fold_routed = np.asarray(
                    data['fold_routed_volume_ml'], dtype=np.float32
                )
                probability_stats = {
                    'local_mean': float('nan'), 'local_max': float('nan'),
                    'proposal_mean': float('nan'), 'proposal_max': float('nan'),
                }
            pairwise = locked_pairwise_dice(fold_masks)
            vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
            union_voxels = int((vote_count > 0).sum())
            disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
            normalized_manifest.append({
                'subject_id': subject_id,
                'dev_split': str(row['dev_split']),
                'path': str(archive_path),
                'sha256': file_sha256(archive_path),
                'native_shape': list(shape),
                'ensemble_local_voxels': int(ensemble_local.sum()),
                'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
                'fold_local_predicted_voxels': [
                    int(mask.sum()) for mask in fold_masks
                ],
                'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
                'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
                'disagreement_union_fraction': float(
                    disagreement / max(union_voxels, 1)
                ),
                'mean_presence_probability': float(fold_presence.mean()),
                'mean_routed_volume_ml': float(fold_routed.mean()),
            })
        prediction_manifest = normalized_manifest
        prediction_gate = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'evaluation_version': LOCKED_EVALUATION_VERSION,
            'code_revision': LOCKED_EVALUATION_CODE_REVISION,
            'prediction_complete': True,
            'prediction_contract': prediction_contract,
            'prediction_contract_sha256': prediction_contract_hash,
            'subjects_predicted': len(prediction_manifest),
            'prediction_manifest': prediction_manifest,
            'ensemble_members': 5,
            'model_constructed_for_inference_only': True,
            'optimizer_constructed': False,
            'optimizer_steps': 0,
            'training_performed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'targets_opened_only_after_this_gate': True,
            'locked_manifest_checks': manifest_checks,
        }
        prediction_checks = {
            'all_46_predictions_present': len(prediction_manifest) == 46,
            'subjects_unique': len({
                item['subject_id'] for item in prediction_manifest
            }) == 46,
            'archives_stable': all(
                Path(item['path']).is_file()
                and file_sha256(item['path']) == item['sha256']
                for item in prediction_manifest
            ),
            'sources_still_unchanged': all(
                locked_sources_unchanged(source_records).values()
            ),
            'locked_manifest_checks_all_true': bool(
                manifest_checks and all(manifest_checks.values())
            ),
        }
        locked_checks_or_raise(
            prediction_checks, 'Target-blind prediction commit failed'
        )
        prediction_gate['checks'] = prediction_checks
        atomic_write_json(prediction_gate, LOCKED_PREDICTION_GATE_PATH)
        del models
        gc.collect()
        torch.cuda.empty_cache()

    
    
    print('\nTarget-blind prediction gate committed. Loading both locked target sets.')
    target_cache = {}
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
        target, soft, spacing = locked_load_target(row, shape)
        target_cache[subject_id] = {
            'target': target,
            'soft_nonzero_voxels': int((soft > 0).sum()),
            'spacing': spacing,
        }
    if len(target_cache) != 46:
        raise RuntimeError('Both locked target sets were not loaded completely.')

    case_rows = []
    component_rows = []
    member_rows = []
    diversity_rows = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
            fold_masks = [
                locked_unpack_mask(bits, shape)
                for bits in data['fold_local_masks_packed']
            ]
            ensemble_local = locked_unpack_mask(
                data['ensemble_local_mask_packed'], shape
            )
            ensemble_proposal = locked_unpack_mask(
                data['ensemble_proposal_mask_packed'], shape
            )
            fold_presence = np.asarray(
                data['fold_presence_probabilities'], dtype=np.float32
            )
            fold_routed = np.asarray(
                data['fold_routed_volume_ml'], dtype=np.float32
            )
        target = target_cache[subject_id]['target']
        spacing = target_cache[subject_id]['spacing']
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
        local_metrics = locked_mask_metrics(
            ensemble_local, target, spacing, 'local'
        )
        proposal_metrics = locked_mask_metrics(
            ensemble_proposal, target, spacing, 'proposal'
        )
        case_record = {
            'subject_id': subject_id,
            'dev_split': split,
            'lesion_volume_bin': lesion_bin,
            'hard_metric_eligible': bool(hard_eligible),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': local_metrics['target_voxels'],
            'predicted_voxels': local_metrics['predicted_voxels'],
            'intersection_voxels': local_metrics['intersection_voxels'],
            'dice': local_metrics['dice'],
            'voxel_precision': local_metrics['voxel_precision'],
            'voxel_recall': local_metrics['voxel_recall'],
            'target_volume_ml': local_metrics['target_voxels'] * voxel_volume_ml,
            'predicted_volume_ml': (
                local_metrics['predicted_voxels'] * voxel_volume_ml
            ),
            'lesion_true_positive': local_metrics['lesion_true_positive'],
            'lesion_false_positive': local_metrics['lesion_false_positive'],
            'lesion_false_negative': local_metrics['lesion_false_negative'],
            'component_true_positive': local_metrics['component_true_positive'],
            'component_total': local_metrics['component_total'],
            'small_component_true_positive': (
                local_metrics['small_component_true_positive']
            ),
            'small_component_total': local_metrics['small_component_total'],
            'proposal_predicted_voxels': proposal_metrics['predicted_voxels'],
            'proposal_dice': proposal_metrics['dice'],
            'proposal_component_true_positive': (
                proposal_metrics['component_true_positive']
            ),
            'proposal_component_total': proposal_metrics['component_total'],
            'mean_presence_probability': float(fold_presence.mean()),
            'mean_routed_volume_ml': float(fold_routed.mean()),
            'soft_nonzero_voxels': target_cache[subject_id][
                'soft_nonzero_voxels'
            ],
            'prediction_archive_path': str(archive_path),
            'prediction_archive_sha256': file_sha256(archive_path),
        }
        for fold, mask in zip(EXPECTED_ENSEMBLE_FOLDS, fold_masks):
            fold_metrics = locked_mask_metrics(mask, target, spacing, 'local')
            case_record[f'fold_{fold}_dice'] = fold_metrics['dice']
            member_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'development_fold_model': fold,
                'hard_metric_eligible': bool(hard_eligible),
                'lesion_volume_bin': lesion_bin,
                'target_voxels': fold_metrics['target_voxels'],
                'predicted_voxels': fold_metrics['predicted_voxels'],
                'intersection_voxels': fold_metrics['intersection_voxels'],
                'dice': fold_metrics['dice'],
                'target_volume_ml': fold_metrics['target_voxels'] * voxel_volume_ml,
                'predicted_volume_ml': (
                    fold_metrics['predicted_voxels'] * voxel_volume_ml
                ),
                'lesion_true_positive': fold_metrics['lesion_true_positive'],
                'lesion_false_positive': fold_metrics['lesion_false_positive'],
                'lesion_false_negative': fold_metrics['lesion_false_negative'],
                'component_true_positive': fold_metrics['component_true_positive'],
                'component_total': fold_metrics['component_total'],
                'small_component_true_positive': (
                    fold_metrics['small_component_true_positive']
                ),
                'small_component_total': fold_metrics['small_component_total'],
            })
        pairwise = locked_pairwise_dice(fold_masks)
        vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
        union = int((vote_count > 0).sum())
        disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
        diversity_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
            'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
            'maximum_pairwise_fold_mask_dice': float(np.max(pairwise)),
            'prediction_union_voxels': union,
            'prediction_disagreement_voxels': disagreement,
            'disagreement_union_fraction': disagreement / max(union, 1),
            'ensemble_dice': local_metrics['dice'],
        })
        for component in local_metrics['components']:
            proposal_component = next(
                item for item in proposal_metrics['components']
                if int(item['component_id']) == int(component['component_id'])
            )
            component_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'component_id': int(component['component_id']),
                'component_tier': component['component_tier'],
                'component_voxels': int(component['component_voxels']),
                'component_volume_ml': float(component['component_volume_ml']),
                'local_maximum_iou': float(component['maximum_component_iou']),
                'local_detected_iou_ge_0p10': bool(
                    float(component['maximum_component_iou']) >= 0.10
                ),
                'proposal_maximum_iou': float(
                    proposal_component['maximum_component_iou']
                ),
                'proposal_detected_iou_ge_0p05': bool(
                    float(proposal_component['maximum_component_iou']) >= 0.05
                ),
            })
        case_rows.append(case_record)

    case_df = pd.DataFrame(case_rows).sort_values(['dev_split', 'subject_id'])
    component_df = pd.DataFrame(component_rows)
    member_case_df = pd.DataFrame(member_rows)
    diversity_df = pd.DataFrame(diversity_rows)
    split_summary_rows = []
    member_summary_rows = []
    bin_rows = []
    bootstrap_rows = []
    split_summaries = {}
    for split in ['val', 'test']:
        split_frame = case_df[case_df['dev_split'] == split].copy()
        summary = locked_aggregate_case_rows(split_frame)
        summary['dev_split'] = split
        split_summaries[split] = summary
        split_summary_rows.append(summary)
        bootstrap_rows.extend(locked_bootstrap_split(
            split_frame, split, LOCKED_BOOTSTRAP_REPLICATES,
            LOCKED_BOOTSTRAP_SEED + (0 if split == 'val' else 1),
        ))
        for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
            bin_frame = split_frame[
                split_frame['lesion_volume_bin'] == lesion_bin
            ].copy()
            if bin_frame.empty:
                continue
            eligible_bin = bin_frame[
                bin_frame['hard_metric_eligible'].astype(bool)
            ]
            positive_bin = eligible_bin[
                eligible_bin['target_voxels'].astype(int) > 0
            ]
            bin_rows.append({
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'subjects': int(len(bin_frame)),
                'positive_subjects': int(len(positive_bin)),
                'mean_dice': float(positive_bin['dice'].mean())
                if len(positive_bin) else float('nan'),
                'median_dice': float(positive_bin['dice'].median())
                if len(positive_bin) else float('nan'),
                'pooled_global_dice': float(
                    2.0 * positive_bin['intersection_voxels'].sum()
                    / max(
                        positive_bin['predicted_voxels'].sum()
                        + positive_bin['target_voxels'].sum(), 1,
                    )
                ) if len(positive_bin) else float('nan'),
                'mean_target_volume_ml': float(
                    positive_bin['target_volume_ml'].mean()
                ) if len(positive_bin) else 0.0,
                'mean_predicted_volume_ml': float(
                    eligible_bin['predicted_volume_ml'].mean()
                ) if len(eligible_bin) else 0.0,
            })
        for fold in EXPECTED_ENSEMBLE_FOLDS:
            member_frame = member_case_df[
                (member_case_df['dev_split'] == split)
                & (member_case_df['development_fold_model'] == fold)
            ].copy()
            member_summary = locked_aggregate_case_rows(member_frame)
            member_summary.update({
                'dev_split': split,
                'model': f'fold_{fold}_ema',
                'development_fold_model': fold,
            })
            member_summary_rows.append(member_summary)
        ensemble_member_summary = summary.copy()
        ensemble_member_summary.update({
            'model': 'fivefold_probability_ensemble',
            'development_fold_model': -1,
        })
        member_summary_rows.append(ensemble_member_summary)

    split_summary_df = pd.DataFrame(split_summary_rows)
    bin_df = pd.DataFrame(bin_rows)
    member_summary_df = pd.DataFrame(member_summary_rows)
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    prediction_audit_df = pd.DataFrame(prediction_manifest)
    source_manifest_df = pd.DataFrame(source_records)
    oof_reference = policy['primary_oof_metrics']
    generalization_rows = []
    metric_mapping = {
        'positive_mean_dice': 'positive_case_mean_dice',
        'positive_median_dice': 'positive_case_median_dice',
        'positive_global_dice': 'pooled_global_dice',
        'lesion_f1': 'lesion_f1',
        'tiny_small_subject_mean_dice': (
            'tiny_small_subject_weighted_mean_dice'
        ),
        'component_recall': 'component_recall',
        'small_component_recall': 'small_component_recall',
    }
    for split in ['val', 'test']:
        for locked_metric, oof_metric in metric_mapping.items():
            locked_value = float(split_summaries[split][locked_metric])
            oof_value = float(oof_reference[oof_metric])
            generalization_rows.append({
                'dev_split': split,
                'metric': locked_metric,
                'development_oof_value': oof_value,
                'locked_value': locked_value,
                'absolute_change_from_oof': locked_value - oof_value,
                'relative_change_from_oof': (
                    (locked_value - oof_value) / abs(oof_value)
                    if abs(oof_value) > 1e-12 else float('nan')
                ),
                'interpretation': (
                    'confirmatory_transfer_diagnostic_not_a_tuning_signal'
                ),
            })
    generalization_df = pd.DataFrame(generalization_rows)
    ensemble_gain = {}
    for split in ['val', 'test']:
        members = member_summary_df[
            (member_summary_df['dev_split'] == split)
            & (member_summary_df['development_fold_model'] >= 0)
        ]
        ensemble_gain[split] = {
            'mean_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_mean_dice']
                - members['positive_mean_dice'].mean()
            ),
            'global_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_global_dice']
                - members['positive_global_dice'].mean()
            ),
            'lesion_f1_gain_over_mean_member': float(
                split_summaries[split]['lesion_f1']
                - members['lesion_f1'].mean()
            ),
        }

    final_checks = {
        **{
            f'locked_manifest_{name}': bool(passed)
            for name, passed in manifest_checks.items()
        },
        'prediction_gate_complete': load_json(
            LOCKED_PREDICTION_GATE_PATH
        ).get('prediction_complete') is True,
        'prediction_contract_matches': load_json(
            LOCKED_PREDICTION_GATE_PATH
        ).get('prediction_contract_sha256') == prediction_contract_hash,
        'all_46_cases_evaluated': len(case_df) == 46,
        'validation_has_23_cases': int(
            (case_df['dev_split'] == 'val').sum()
        ) == 23,
        'test_has_23_cases': int(
            (case_df['dev_split'] == 'test').sum()
        ) == 23,
        'all_subjects_unique': case_df['subject_id'].is_unique,
        'all_case_metrics_finite': bool(np.isfinite(case_df[[
            'dice', 'voxel_precision', 'voxel_recall',
            'target_volume_ml', 'predicted_volume_ml',
        ]].to_numpy(dtype=float)).all()),
        'both_split_summaries_finite': bool(np.isfinite(
            split_summary_df.select_dtypes(include=[np.number]).to_numpy()
        ).all()),
        'bootstrap_complete': len(bootstrap_df) == 20,
        'bootstrap_intervals_finite': bool(np.isfinite(bootstrap_df[[
            'point_estimate', 'ci_95_lower', 'ci_95_upper',
            'bootstrap_standard_error',
        ]].to_numpy(dtype=float)).all()),
        'five_members_plus_ensemble_per_split': len(member_summary_df) == 12,
        'generalization_rows_complete': len(generalization_df) == 14,
        'generalization_core_values_finite': bool(np.isfinite(
            generalization_df[[
                'development_oof_value', 'locked_value',
                'absolute_change_from_oof',
            ]].to_numpy(dtype=float)
        ).all()),
        'frozen_local_threshold_unchanged': local_threshold == 0.40,
        'frozen_proposal_threshold_unchanged': proposal_threshold == 0.35,
        'postprocessing_remained_none': policy['postprocessing'] == 'none',
        'tta_remained_none': policy['test_time_augmentation'] == 'none',
        'architecture_unchanged': True,
        'checkpoints_unchanged': all(
            locked_sources_unchanged(source_records).values()
        ),
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'post_freeze_tuning_not_performed': True,
        'validation_and_test_evaluated_in_one_frozen_execution': True,
    }
    locked_checks_or_raise(final_checks, 'Final locked evaluation failed')

    atomic_write_csv(source_manifest_df, LOCKED_SOURCE_MANIFEST_PATH)
    atomic_write_csv(prediction_audit_df, LOCKED_PREDICTION_AUDIT_PATH)
    atomic_write_csv(case_df, LOCKED_CASE_METRICS_PATH)
    atomic_write_csv(component_df, LOCKED_COMPONENT_METRICS_PATH)
    atomic_write_csv(split_summary_df, LOCKED_SPLIT_SUMMARY_PATH)
    atomic_write_csv(bin_df, LOCKED_BIN_METRICS_PATH)
    atomic_write_csv(member_summary_df, LOCKED_MEMBER_METRICS_PATH)
    atomic_write_csv(diversity_df, LOCKED_DIVERSITY_PATH)
    atomic_write_csv(bootstrap_df, LOCKED_BOOTSTRAP_PATH)
    atomic_write_csv(generalization_df, LOCKED_GENERALIZATION_PATH)
    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'selected_architecture': policy['selected_architecture'],
        'ensemble_members': 5,
        'checkpoint_weight_state': 'ema_model_state',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'development_oof_reference': oof_reference,
        'ensemble_gain_over_mean_member': ensemble_gain,
        'bootstrap_replicates': LOCKED_BOOTSTRAP_REPLICATES,
        'metric_scope': (
            'hard-metric-eligible positive cases for Dice; hard-metric-eligible '
            'cases for lesion detection; thresholds frozen from development OOF'
        ),
        'validation_used_for_tuning': False,
        'test_used_for_tuning': False,
        'post_freeze_tuning_performed': False,
        'scientific_interpretation': (
            'Validation and test are confirmatory locked evaluations. The '
            'development OOF estimate remains separately reported and is not '
            'pooled with either locked split as a primary performance estimate.'
        ),
    }
    atomic_write_json(report, LOCKED_REPORT_PATH)
    output_paths = {
        'target_blind_prediction_gate': LOCKED_PREDICTION_GATE_PATH,
        'source_artifact_manifest': LOCKED_SOURCE_MANIFEST_PATH,
        'prediction_audit': LOCKED_PREDICTION_AUDIT_PATH,
        'case_metrics': LOCKED_CASE_METRICS_PATH,
        'component_metrics': LOCKED_COMPONENT_METRICS_PATH,
        'split_summary': LOCKED_SPLIT_SUMMARY_PATH,
        'lesion_bin_metrics': LOCKED_BIN_METRICS_PATH,
        'ensemble_member_metrics': LOCKED_MEMBER_METRICS_PATH,
        'ensemble_diversity_metrics': LOCKED_DIVERSITY_PATH,
        'bootstrap_intervals': LOCKED_BOOTSTRAP_PATH,
        'oof_to_locked_generalization': LOCKED_GENERALIZATION_PATH,
        'evaluation_report': LOCKED_REPORT_PATH,
    }
    output_manifest = {
        name: {'path': str(path), 'sha256': file_sha256(path)}
        for name, path in output_paths.items()
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'evaluation_complete': True,
        'evaluated_splits': ['val', 'test'],
        'subjects_evaluated': {'val': 23, 'test': 23},
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'ensemble_members': 5,
        'ensemble_checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_weight_state': 'ema_model_state',
        'ensemble_probability_rule': (
            'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'prediction_contract_sha256': prediction_contract_hash,
        'prediction_gate_path': str(LOCKED_PREDICTION_GATE_PATH),
        'prediction_gate_sha256': file_sha256(LOCKED_PREDICTION_GATE_PATH),
        'source_artifact_manifest': source_records,
        'output_artifact_manifest': output_manifest,
        'final_checks': final_checks,
        'locked_validation_targets_read': True,
        'test_targets_read': True,
        'targets_read_only_after_target_blind_prediction_commit': True,
        'model_constructed_for_inference_only': True,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'post_freeze_tuning_performed': False,
        'further_model_or_threshold_tuning_authorized': False,
        'next_action': (
            'Report the frozen OOF, locked validation, and locked test results; '
            'perform no further model selection or threshold tuning.'
        ),
    }
    atomic_write_json(gate, LOCKED_FINAL_GATE_PATH)

    print('\nLocked split summary:')
    display(split_summary_df[[
        'dev_split', 'subjects', 'positive_subjects',
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]])
    print('\nLocked lesion-volume-bin metrics:')
    display(bin_df)
    print('\nLocked 95% descriptive bootstrap intervals:')
    display(bootstrap_df)
    print('\nEnsemble versus member models:')
    display(member_summary_df[[
        'dev_split', 'model', 'positive_mean_dice',
        'positive_global_dice', 'lesion_f1',
        'tiny_small_subject_mean_dice',
    ]])
    print('\nDevelopment OOF to locked-split generalization:')
    display(generalization_df[[
        'dev_split', 'metric', 'development_oof_value', 'locked_value',
        'absolute_change_from_oof',
    ]])
    print('\n' + '=' * 80)
    print('TRIAL8 FINAL LOCKED VALIDATION/TEST ENSEMBLE EVALUATION PASSED')
    print('=' * 80)
    for split in ['val', 'test']:
        item = split_summaries[split]
        print(
            f'{split.upper():10s} mean / median / global Dice: '
            f'{item["positive_mean_dice"]:.4f} / '
            f'{item["positive_median_dice"]:.4f} / '
            f'{item["positive_global_dice"]:.4f}'
        )
        print(
            f'{split.upper():10s} lesion F1 / component recall: '
            f'{item["lesion_f1"]:.4f} / '
            f'{item["component_recall"]:.4f}'
        )
    print('Frozen local / proposal threshold:  0.40 / 0.35')
    print('Postprocessing / TTA:               none / none')
    print('Post-freeze tuning performed:       False')
    print('Further tuning authorized:          False')
    print(f'Final report:                       {LOCKED_REPORT_PATH}')
    print(f'Final gate:                         {LOCKED_FINAL_GATE_PATH}')
    print('\nNext: report results and limitations; do not tune on these splits.')


run_locked_fivefold_evaluation()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )















LOCKED_EVALUATION_VERSION = 'v3_final_locked_val_test_metric_reconciliation'
LOCKED_EVALUATION_CODE_REVISION = (
    'r3_target_derived_metric_eligibility_reuse_frozen_r2_predictions'
)
SOURCE_PREDICTION_VERSION = 'v2_final_locked_val_test_fivefold_ensemble_once'
SOURCE_PREDICTION_CODE_REVISION = (
    'r2_locked_input_contract_training_lock_semantics'
)
OOF_FREEZE_VERSION = 'v1_final_fivefold_oof_threshold_freeze'
EXPECTED_OOF_AGGREGATION_VERSION = OOF_FREEZE_VERSION
EXPECTED_OOF_CODE_REVISION = (
    'r1_exact_sufficient_statistics_locked_policy'
)
EXPECTED_ENSEMBLE_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_LOCKED_SPLITS = {'val': 23, 'test': 23}
EXPECTED_FROZEN_LOCAL_THRESHOLD = 0.40
EXPECTED_FROZEN_PROPOSAL_THRESHOLD = 0.35
LOCKED_BOOTSTRAP_REPLICATES = 10_000
LOCKED_BOOTSTRAP_SEED = 20261012

OOF_FREEZE_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / OOF_FREEZE_VERSION
)
OOF_DEPLOYMENT_POLICY_PATH = OOF_FREEZE_ROOT / 'frozen_deployment_policy.json'
OOF_AGGREGATION_GATE_PATH = OOF_FREEZE_ROOT / 'oof_threshold_freeze_gate.json'

LOCKED_OUTPUT_ROOT = (
    PROJECT_ROOT / 'evaluation' / MODEL_VERSION / LOCKED_EVALUATION_VERSION
)
SOURCE_PREDICTION_ROOT = (
    PROJECT_ROOT / 'evaluation' / MODEL_VERSION / SOURCE_PREDICTION_VERSION
)
LOCKED_PREDICTION_ROOT = SOURCE_PREDICTION_ROOT / 'target_blind_predictions'
LOCKED_PREDICTION_INDEX_PATH = SOURCE_PREDICTION_ROOT / 'prediction_index.json'
LOCKED_PREDICTION_GATE_PATH = (
    SOURCE_PREDICTION_ROOT / 'target_blind_prediction_gate.json'
)
SOURCE_FLAWED_FINAL_GATE_PATH = SOURCE_PREDICTION_ROOT / 'locked_evaluation_gate.json'
LOCKED_SOURCE_MANIFEST_PATH = LOCKED_OUTPUT_ROOT / 'source_artifact_manifest.csv'
LOCKED_PREDICTION_AUDIT_PATH = LOCKED_OUTPUT_ROOT / 'prediction_audit.csv'
LOCKED_TARGET_POLICY_AUDIT_PATH = (
    LOCKED_OUTPUT_ROOT / 'target_derived_policy_audit.csv'
)
LOCKED_CASE_METRICS_PATH = LOCKED_OUTPUT_ROOT / 'locked_case_metrics.csv'
LOCKED_COMPONENT_METRICS_PATH = (
    LOCKED_OUTPUT_ROOT / 'locked_component_metrics.csv'
)
LOCKED_SPLIT_SUMMARY_PATH = LOCKED_OUTPUT_ROOT / 'locked_split_summary.csv'
LOCKED_BIN_METRICS_PATH = LOCKED_OUTPUT_ROOT / 'locked_lesion_bin_metrics.csv'
LOCKED_MEMBER_METRICS_PATH = (
    LOCKED_OUTPUT_ROOT / 'ensemble_member_metrics.csv'
)
LOCKED_DIVERSITY_PATH = LOCKED_OUTPUT_ROOT / 'ensemble_diversity_metrics.csv'
LOCKED_BOOTSTRAP_PATH = LOCKED_OUTPUT_ROOT / 'locked_bootstrap_intervals.csv'
LOCKED_GENERALIZATION_PATH = (
    LOCKED_OUTPUT_ROOT / 'oof_to_locked_generalization_summary.csv'
)
LOCKED_REPORT_PATH = LOCKED_OUTPUT_ROOT / 'locked_evaluation_report.json'
LOCKED_FINAL_GATE_PATH = LOCKED_OUTPUT_ROOT / 'locked_evaluation_gate.json'


def locked_checks_or_raise(checks, context):
    failed = [name for name, passed in checks.items() if not bool(passed)]
    if failed:
        raise RuntimeError(context + ': ' + ', '.join(failed))


def locked_canonical_hash(record):
    payload = json.dumps(
        to_builtin(record), sort_keys=True, separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def locked_source_record(path, role, development_fold=None):
    path = Path(path)
    stat = path.stat()
    return {
        'role': str(role),
        'development_fold': (
            int(development_fold) if development_fold is not None else -1
        ),
        'path': str(path),
        'sha256': file_sha256(path),
        'size_bytes': int(stat.st_size),
        'mtime_ns': int(stat.st_mtime_ns),
    }


def locked_sources_unchanged(records):
    checks = {}
    for index, record in enumerate(records):
        path = Path(record['path'])
        stat = path.stat() if path.exists() else None
        checks[f'source_{index:03d}_{record["role"]}_unchanged'] = bool(
            stat is not None
            and int(stat.st_size) == int(record['size_bytes'])
            and int(stat.st_mtime_ns) == int(record['mtime_ns'])
            and file_sha256(path) == record['sha256']
        )
    return checks


def locked_manifest_input_contract_checks(locked_rows):
    
    required_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing = sorted(required_columns - set(locked_rows.columns))
    if missing:
        raise RuntimeError(
            'Locked manifest is missing required inference columns: '
            + ', '.join(missing)
        )

    path_columns = [
        'vascufate_v2_npz_path', 'trial8_graph_sidecar_path',
        'trial8_clinical_sidecar_path',
    ]
    paths_declared = locked_rows[path_columns].apply(
        lambda column: column.map(
            lambda value: isinstance(value, str) and bool(value.strip())
        )
    )
    paths_exist = locked_rows[path_columns].apply(
        lambda column: column.map(
            lambda value: isinstance(value, str) and Path(value).is_file()
        )
    )
    hash_columns = [
        'trial8_npz_sha256', 'trial8_graph_sidecar_sha256',
        'trial8_clinical_sidecar_sha256',
    ]
    hashes_declared = locked_rows[hash_columns].apply(
        lambda column: column.map(
            lambda value: (
                isinstance(value, str)
                and len(value.strip()) == 64
                and all(character in '0123456789abcdef'
                        for character in value.strip().lower())
            )
        )
    )
    atlas_paths_valid = []
    for row in locked_rows.to_dict('records'):
        if as_bool(row['trial8_atlas_enabled']):
            atlas_path = str(row['trial8_atlas_vascular30_path']).strip()
            atlas_paths_valid.append(bool(atlas_path) and Path(atlas_path).is_file())
        else:
            atlas_paths_valid.append(True)

    loader_passed = bool(
        locked_rows['trial8_v2_multimodal_loader_preflight_passed']
        .map(as_bool).all()
    )
    training_locked = not bool(
        locked_rows['trial8_model_training_authorized'].map(as_bool).any()
    )
    inputs_verified = bool(
        loader_passed
        and training_locked
        and paths_declared.to_numpy(dtype=bool).all()
        and paths_exist.to_numpy(dtype=bool).all()
        and hashes_declared.to_numpy(dtype=bool).all()
        and all(atlas_paths_valid)
    )
    return {
        'all_loader_preflights_passed': loader_passed,
        'training_remains_locked_for_all': training_locked,
        'all_model_input_paths_declared': bool(
            paths_declared.to_numpy(dtype=bool).all()
        ),
        'all_model_input_files_exist': bool(
            paths_exist.to_numpy(dtype=bool).all()
        ),
        'all_model_input_hashes_declared': bool(
            hashes_declared.to_numpy(dtype=bool).all()
        ),
        'all_enabled_atlas_paths_exist': bool(all(atlas_paths_valid)),
        'all_target_free_model_inputs_verified': inputs_verified,
    }


def locked_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return np.packbits(mask.reshape(-1), bitorder='little')


def locked_unpack_mask(bits, shape):
    count = int(np.prod(shape))
    values = np.unpackbits(
        np.asarray(bits, dtype=np.uint8), bitorder='little'
    )[:count]
    return values.reshape(tuple(int(v) for v in shape)).astype(bool)


def locked_atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def locked_prediction_path(subject_id):
    return LOCKED_PREDICTION_ROOT / f'{subject_id}_fivefold_ensemble_masks.npz'


def locked_load_target_free_case(row):
    
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    native_shape = tuple(int(v) for v in image.shape[1:])
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(native_shape, dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: inference padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id,
        'row': row,
        'image_canvas': image_canvas,
        'region_canvas': region_canvas,
        'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas,
        'lvo_present': lvo_present,
        'native_shape': native_shape,
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


@torch.no_grad()
def locked_infer_case(model, row, device):
    case = locked_load_target_free_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(model, base_batch)
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][axis] + native_start[axis])
                    for axis in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(model, batch, cache)
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(
                        native_start[axis],
                        min(
                            native_start[axis] + LOCAL_PATCH_SIZE[axis],
                            native_shape[axis],
                        ),
                    )
                    for axis in range(3)
                )
                copied_shape = tuple(
                    item.stop - item.start for item in destination
                )
                source = tuple(
                    slice(0, copied_shape[axis]) for axis in range(3)
                )
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete sliding-window coverage."
        )
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(
            case['pad_start'][axis],
            case['pad_start'][axis] + native_shape[axis],
        )
        for axis in range(3)
    )
    proposal = proposal_canvas[native_slices].astype(np.float32)
    presence = float(torch.sigmoid(
        cache['routed']['lesion_presence_logit'].float()
    )[0].cpu())
    routed_volume_ml = float(torch.expm1(
        cache['routed']['log_volume_prediction'].float()
    )[0].clamp_min(0).cpu())
    del base_batch, cache
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float32),
        'proposal_probability': proposal,
        'presence_probability': presence,
        'routed_volume_ml': routed_volume_ml,
    }


def locked_validate_prediction_archive(path, expected_contract_hash, subject_id):
    path = Path(path)
    if not path.is_file():
        return None
    with np.load(path, allow_pickle=False) as data:
        required = {
            'subject_id', 'dev_split', 'native_shape', 'contract_sha256',
            'fold_ids', 'fold_local_masks_packed',
            'ensemble_local_mask_packed', 'ensemble_proposal_mask_packed',
            'fold_predicted_voxels', 'fold_presence_probabilities',
            'fold_routed_volume_ml', 'ensemble_local_probability_sha256',
            'ensemble_proposal_probability_sha256',
        }
        if not required.issubset(data.files):
            raise RuntimeError(f'{subject_id}: prediction archive is incomplete.')
        observed_subject = scalar_string(data['subject_id'])
        observed_contract = scalar_string(data['contract_sha256'])
        shape = tuple(int(v) for v in np.asarray(data['native_shape']).tolist())
        fold_ids = tuple(int(v) for v in np.asarray(data['fold_ids']).tolist())
        fold_bits = np.asarray(data['fold_local_masks_packed'], dtype=np.uint8)
        local_bits = np.asarray(
            data['ensemble_local_mask_packed'], dtype=np.uint8
        )
        proposal_bits = np.asarray(
            data['ensemble_proposal_mask_packed'], dtype=np.uint8
        )
    expected_bytes = (int(np.prod(shape)) + 7) // 8
    checks = {
        'subject_matches': observed_subject == subject_id,
        'contract_matches': observed_contract == expected_contract_hash,
        'shape_is_three_dimensional': len(shape) == 3,
        'fold_ids_exact': fold_ids == EXPECTED_ENSEMBLE_FOLDS,
        'fold_mask_shape_exact': fold_bits.shape == (5, expected_bytes),
        'ensemble_local_mask_size_exact': local_bits.size == expected_bytes,
        'ensemble_proposal_mask_size_exact': proposal_bits.size == expected_bytes,
    }
    locked_checks_or_raise(
        checks, f'{subject_id}: existing target-blind prediction rejected'
    )
    return {
        'subject_id': subject_id,
        'path': str(path),
        'sha256': file_sha256(path),
        'native_shape': list(shape),
    }


def locked_pairwise_dice(masks):
    values = []
    for left in range(len(masks)):
        for right in range(left + 1, len(masks)):
            denominator = int(masks[left].sum() + masks[right].sum())
            intersection = int((masks[left] & masks[right]).sum())
            values.append(
                2.0 * intersection / denominator if denominator else 1.0
            )
    return values


def locked_prediction_manifest_row(
    subject_id, split, archive_path, shape, fold_masks,
    ensemble_local, ensemble_proposal, local_probability,
    proposal_probability, fold_presence, fold_routed_volume,
):
    pairwise = locked_pairwise_dice(fold_masks)
    vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
    union_voxels = int((vote_count > 0).sum())
    disagreement_voxels = int(((vote_count > 0) & (vote_count < 5)).sum())
    return {
        'subject_id': subject_id,
        'dev_split': split,
        'prediction_path': str(archive_path),
        'prediction_sha256': file_sha256(archive_path),
        'native_shape': list(shape),
        'ensemble_local_voxels': int(ensemble_local.sum()),
        'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
        'fold_local_predicted_voxels': [int(mask.sum()) for mask in fold_masks],
        'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
        'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
        'disagreement_union_fraction': float(
            disagreement_voxels / max(union_voxels, 1)
        ),
        'ensemble_local_probability_mean': float(local_probability.mean()),
        'ensemble_local_probability_maximum': float(local_probability.max()),
        'ensemble_proposal_probability_mean': float(
            proposal_probability.mean()
        ),
        'ensemble_proposal_probability_maximum': float(
            proposal_probability.max()
        ),
        'mean_presence_probability': float(np.mean(fold_presence)),
        'mean_routed_volume_ml': float(np.mean(fold_routed_volume)),
    }


def locked_load_target(row, expected_shape):
    subject_id = str(row['subject_id'])
    path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(path, allow_pickle=False) as data:
        if not {'target_binary', 'target_soft', 'target_spacing'}.issubset(
            data.files
        ):
            raise RuntimeError(f'{subject_id}: locked target arrays unavailable.')
        target_raw = np.asarray(data['target_binary'])
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    target = target_raw.astype(np.uint8, copy=False) > 0
    checks = {
        'target_shape_matches_prediction': tuple(target.shape) == tuple(expected_shape),
        'soft_shape_matches_prediction': tuple(soft.shape) == tuple(expected_shape),
        'binary_target_is_finite': bool(np.isfinite(target_raw).all()),
        'binary_target_contains_only_zero_or_one': bool(
            np.isin(target_raw, [0, 1]).all()
        ),
        'spacing_has_three_values': spacing.shape == (3,),
        'spacing_is_finite_positive': bool(
            np.isfinite(spacing).all() and (spacing > 0).all()
        ),
        'soft_is_finite': bool(np.isfinite(soft).all()),
        'soft_is_nonnegative': bool((soft >= 0).all()),
    }
    locked_checks_or_raise(checks, f'{subject_id}: locked target integrity failed')
    return target, soft, spacing


def locked_derive_target_policy(target, soft, spacing):
    
    target = np.asarray(target, dtype=bool)
    soft = np.asarray(soft, dtype=np.float64)
    spacing = np.asarray(spacing, dtype=np.float64).reshape(-1)
    checks = {
        'target_and_soft_shapes_match': target.shape == soft.shape,
        'spacing_has_three_values': spacing.shape == (3,),
        'spacing_is_finite_positive': bool(
            np.isfinite(spacing).all() and (spacing > 0).all()
        ),
        'soft_is_finite_nonnegative': bool(
            np.isfinite(soft).all() and (soft >= 0).all()
        ),
    }
    locked_checks_or_raise(checks, 'Target-derived policy inputs are invalid')
    hard_voxels = int(target.sum())
    soft_support_voxels = int((soft > 0).sum())
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    hard_volume_ml = float(hard_voxels * voxel_volume_ml)
    soft_volume_ml = float(soft.sum(dtype=np.float64) * voxel_volume_ml)
    if hard_voxels > 0:
        supervision_mode = 'hard_and_soft'
        hard_metric_eligible = True
        effective_volume_ml = hard_volume_ml
    elif soft_support_voxels > 0:
        supervision_mode = 'soft_only_subvoxel'
        hard_metric_eligible = False
        effective_volume_ml = soft_volume_ml
    else:
        supervision_mode = 'true_empty'
        hard_metric_eligible = True
        effective_volume_ml = 0.0
    if supervision_mode == 'true_empty':
        lesion_volume_bin = 'empty'
    elif effective_volume_ml < 1.0:
        lesion_volume_bin = 'tiny'
    elif effective_volume_ml < 5.0:
        lesion_volume_bin = 'small'
    elif effective_volume_ml < 25.0:
        lesion_volume_bin = 'medium'
    elif effective_volume_ml < 75.0:
        lesion_volume_bin = 'large'
    else:
        lesion_volume_bin = 'very_large'
    return {
        'supervision_mode': supervision_mode,
        'hard_metric_eligible': bool(hard_metric_eligible),
        'lesion_volume_bin': lesion_volume_bin,
        'hard_voxels': hard_voxels,
        'soft_support_voxels': soft_support_voxels,
        'voxel_volume_ml': voxel_volume_ml,
        'hard_target_volume_ml': hard_volume_ml,
        'soft_target_volume_ml': soft_volume_ml,
        'effective_target_volume_ml': float(effective_volume_ml),
    }


def locked_mask_metrics(prediction, target, spacing, probability_role='local'):
    intersection = int((prediction & target).sum())
    predicted = int(prediction.sum())
    target_voxels = int(target.sum())
    denominator = predicted + target_voxels
    dice = 2.0 * intersection / denominator if denominator else 1.0
    lesion_tp, lesion_fp, lesion_fn = pilot_lesion_counts(prediction, target)
    iou_threshold = 0.05 if probability_role == 'proposal' else 0.10
    components = pilot_target_component_iou_records(
        prediction, target, spacing
    ) if target_voxels > 0 else []
    component_tp = sum(
        float(item['maximum_component_iou']) >= iou_threshold
        for item in components
    )
    small_components = [
        item for item in components
        if item['component_tier'] in {
            'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
        }
    ]
    small_tp = sum(
        float(item['maximum_component_iou']) >= iou_threshold
        for item in small_components
    )
    return {
        'intersection_voxels': intersection,
        'predicted_voxels': predicted,
        'target_voxels': target_voxels,
        'dice': float(dice),
        'voxel_precision': intersection / max(predicted, 1),
        'voxel_recall': intersection / max(target_voxels, 1),
        'lesion_true_positive': int(lesion_tp),
        'lesion_false_positive': int(lesion_fp),
        'lesion_false_negative': int(lesion_fn),
        'component_true_positive': int(component_tp),
        'component_total': int(len(components)),
        'small_component_true_positive': int(small_tp),
        'small_component_total': int(len(small_components)),
        'components': components,
    }


def locked_aggregate_case_rows(frame):
    eligible = frame[frame['hard_metric_eligible'].astype(bool)].copy()
    positive = eligible[eligible['target_voxels'].astype(int) > 0].copy()
    tiny_small = positive[positive['lesion_volume_bin'].isin(['tiny', 'small'])]
    intersection = int(positive['intersection_voxels'].sum())
    predicted = int(positive['predicted_voxels'].sum())
    target = int(positive['target_voxels'].sum())
    lesion_tp = int(eligible['lesion_true_positive'].sum())
    lesion_fp = int(eligible['lesion_false_positive'].sum())
    lesion_fn = int(eligible['lesion_false_negative'].sum())
    precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * precision * recall / max(precision + recall, 1e-12)
    )
    component_tp = int(positive['component_true_positive'].sum())
    component_total = int(positive['component_total'].sum())
    small_tp = int(positive['small_component_true_positive'].sum())
    small_total = int(positive['small_component_total'].sum())
    target_volume = float(positive['target_volume_ml'].sum())
    predicted_volume = float(positive['predicted_volume_ml'].sum())
    return {
        'subjects': int(len(frame)),
        'hard_metric_eligible_subjects': int(len(eligible)),
        'positive_subjects': int(len(positive)),
        'positive_mean_dice': float(positive['dice'].mean()) if len(positive) else 0.0,
        'positive_median_dice': float(positive['dice'].median()) if len(positive) else 0.0,
        'positive_global_dice': float(
            2.0 * intersection / max(predicted + target, 1)
        ),
        'lesion_precision': float(precision),
        'lesion_recall': float(recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': component_tp / max(component_total, 1),
        'small_component_recall': small_tp / max(small_total, 1),
        'tiny_small_subject_mean_dice': float(
            tiny_small['dice'].mean()
        ) if len(tiny_small) else 0.0,
        'predicted_to_target_volume_ratio': (
            predicted_volume / max(target_volume, 1e-12)
        ),
        'empty_case_mean_false_positive_volume_ml': float(
            eligible.loc[
                eligible['target_voxels'].astype(int) == 0,
                'predicted_volume_ml',
            ].mean()
        ) if bool((eligible['target_voxels'].astype(int) == 0).any()) else 0.0,
        'intersection_voxels': intersection,
        'predicted_voxels_positive_cases': predicted,
        'target_voxels_positive_cases': target,
        'lesion_true_positive': lesion_tp,
        'lesion_false_positive': lesion_fp,
        'lesion_false_negative': lesion_fn,
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


def locked_bootstrap_split(frame, split, replicates, seed):
    frame = frame.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    metric_names = [
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]
    observed = locked_aggregate_case_rows(frame)
    samples = {name: np.empty(replicates, dtype=np.float64) for name in metric_names}
    for replicate in range(replicates):
        indices = rng.integers(0, len(frame), size=len(frame))
        summary = locked_aggregate_case_rows(frame.iloc[indices])
        for name in metric_names:
            samples[name][replicate] = float(summary[name])
    rows = []
    for name in metric_names:
        values = samples[name]
        rows.append({
            'dev_split': split,
            'metric': name,
            'point_estimate': float(observed[name]),
            'ci_95_lower': float(np.quantile(values, 0.025)),
            'ci_95_upper': float(np.quantile(values, 0.975)),
            'bootstrap_standard_error': float(values.std(ddof=1)),
            'bootstrap_replicates': int(replicates),
            'bootstrap_seed': int(seed),
            'method': (
                'subject_level_nonparametric_percentile_fixed_frozen_policy'
            ),
        })
    return rows


def locked_existing_final_is_valid():
    if not LOCKED_FINAL_GATE_PATH.exists():
        return None
    gate = load_json(LOCKED_FINAL_GATE_PATH)
    expected_outputs = {
        'target_blind_prediction_gate', 'source_artifact_manifest',
        'prediction_audit', 'target_policy_audit', 'case_metrics',
        'component_metrics',
        'split_summary', 'lesion_bin_metrics', 'ensemble_member_metrics',
        'ensemble_diversity_metrics', 'bootstrap_intervals',
        'oof_to_locked_generalization', 'evaluation_report',
    }
    output_manifest = gate.get('output_artifact_manifest', {})
    source_manifest = gate.get('source_artifact_manifest', [])
    checks = {
        'evaluation_complete': gate.get('evaluation_complete') is True,
        'version_matches': gate.get('evaluation_version') == LOCKED_EVALUATION_VERSION,
        'revision_matches': gate.get('code_revision') == LOCKED_EVALUATION_CODE_REVISION,
        'local_threshold_matches': math.isclose(
            float(gate.get('frozen_local_threshold', float('nan'))),
            EXPECTED_FROZEN_LOCAL_THRESHOLD, rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_matches': math.isclose(
            float(gate.get('frozen_proposal_threshold', float('nan'))),
            EXPECTED_FROZEN_PROPOSAL_THRESHOLD, rel_tol=0.0, abs_tol=1e-12,
        ),
        'validation_and_test_evaluated_together': (
            gate.get('evaluated_splits') == ['val', 'test']
        ),
        'no_tuning': gate.get('post_freeze_tuning_performed') is False,
        'output_manifest_schema_exact': set(output_manifest) == expected_outputs,
        'source_manifest_complete': len(source_manifest) == 69,
        'source_predictions_reused': gate.get('source_predictions_reused') is True,
        'model_not_constructed': (
            gate.get('model_constructed_for_metric_reconciliation') is False
        ),
        'target_policy_derived_after_prediction_commit': (
            gate.get('target_policy_derived_after_prediction_commit') is True
        ),
        'recorded_final_checks_all_true': bool(
            gate.get('final_checks')
            and all(gate.get('final_checks', {}).values())
        ),
        'artifacts_stable': all(
            Path(record['path']).is_file()
            and file_sha256(Path(record['path'])) == record['sha256']
            for record in output_manifest.values()
        ),
        'sources_stable': all(
            Path(record['path']).is_file()
            and file_sha256(Path(record['path'])) == record['sha256']
            for record in source_manifest
        ),
    }
    locked_checks_or_raise(
        checks,
        'Existing locked-evaluation gate failed integrity verification; '
        'refusing to rerun or overwrite',
    )
    return gate


def run_locked_fivefold_evaluation():
    existing = locked_existing_final_is_valid()
    if existing is not None:
        summaries = existing['split_summaries']
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL LOCKED VALIDATION/TEST EVALUATION ALREADY COMPLETE')
        print('=' * 80)
        for split in ['val', 'test']:
            item = summaries[split]
            print(
                f'{split.upper():10s} mean/global Dice:             '
                f'{item["positive_mean_dice"]:.4f} / '
                f'{item["positive_global_dice"]:.4f}'
            )
        print(f'Final gate: {LOCKED_FINAL_GATE_PATH}')
        print('No checkpoint, prediction, target, or result artifact was rewritten.')
        return

    required_paths = [
        OOF_DEPLOYMENT_POLICY_PATH, OOF_AGGREGATION_GATE_PATH,
        MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
        MULTIMODAL_GATE_PATH,
    ]
    missing = [str(path) for path in required_paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(
            'Locked-evaluation prerequisites are missing:\n' + '\n'.join(missing)
        )
    policy = load_json(OOF_DEPLOYMENT_POLICY_PATH)
    oof_gate = load_json(OOF_AGGREGATION_GATE_PATH)
    policy_hash = file_sha256(OOF_DEPLOYMENT_POLICY_PATH)
    source_records = [
        locked_source_record(OOF_DEPLOYMENT_POLICY_PATH, 'deployment_policy'),
        locked_source_record(OOF_AGGREGATION_GATE_PATH, 'oof_freeze_gate'),
        locked_source_record(MULTIMODAL_MANIFEST_PATH, 'multimodal_manifest'),
        locked_source_record(FOLD_MANIFEST_PATH, 'fold_manifest'),
        locked_source_record(GRAPH_SCHEMA_PATH, 'graph_schema'),
        locked_source_record(MULTIMODAL_GATE_PATH, 'multimodal_gate'),
    ]
    checkpoints = sorted(
        policy.get('ensemble_checkpoints', []),
        key=lambda item: int(item['development_fold']),
    )
    for record in checkpoints:
        source_records.append(locked_source_record(
            record['best_checkpoint_path'], 'best_ema_checkpoint',
            record['development_fold'],
        ))
        graph_scaler = Path(resolve_gate_mapping(
            multimodal_gate['graph_scaler_paths'], record['development_fold']
        ))
        clinical_scaler = Path(resolve_gate_mapping(
            multimodal_gate['clinical_scaler_paths'], record['development_fold']
        ))
        source_records.append(locked_source_record(
            graph_scaler, 'fold_graph_scaler', record['development_fold']
        ))
        source_records.append(locked_source_record(
            clinical_scaler, 'fold_clinical_scaler', record['development_fold']
        ))
    local_threshold = float(policy.get('frozen_local_threshold', float('nan')))
    proposal_threshold = float(
        policy.get('frozen_proposal_threshold', float('nan'))
    )
    checkpoint_folds = tuple(
        int(record['development_fold']) for record in checkpoints
    )
    upstream_checks = {
        'oof_aggregation_complete': oof_gate.get('aggregation_complete') is True,
        'locked_evaluation_authorized': (
            oof_gate.get('locked_evaluation_authorized') is True
            and policy.get('locked_evaluation_authorized') is True
        ),
        'aggregation_version_matches': (
            oof_gate.get('aggregation_version') == EXPECTED_OOF_AGGREGATION_VERSION
            and policy.get('aggregation_version') == EXPECTED_OOF_AGGREGATION_VERSION
        ),
        'aggregation_code_revision_matches': (
            oof_gate.get('code_revision') == EXPECTED_OOF_CODE_REVISION
            and policy.get('code_revision') == EXPECTED_OOF_CODE_REVISION
        ),
        'policy_hash_matches_oof_output_manifest': (
            oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('path') == str(OOF_DEPLOYMENT_POLICY_PATH)
            and oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('sha256') == policy_hash
        ),
        'oof_final_checks_all_true': bool(
            oof_gate.get('final_checks')
            and all(oof_gate.get('final_checks', {}).values())
        ),
        'five_checkpoints_exact': checkpoint_folds == EXPECTED_ENSEMBLE_FOLDS,
        'ema_weight_state_exact': all(
            record.get('weight_state') == 'ema_model_state'
            for record in checkpoints
        ) and policy.get('checkpoint_weight_state') == 'ema_model_state',
        'checkpoint_hashes_match_policy': all(
            file_sha256(record['best_checkpoint_path'])
            == record['best_checkpoint_sha256']
            for record in checkpoints
        ),
        'checkpoint_parameters_match': all(
            int(record['parameter_count']) == EXPECTED_V22_PARAMETERS
            for record in checkpoints
        ),
        'local_threshold_exact': math.isclose(
            local_threshold, EXPECTED_FROZEN_LOCAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_exact': math.isclose(
            proposal_threshold, EXPECTED_FROZEN_PROPOSAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'probability_ensemble_rule_exact': (
            policy.get('ensemble_probability_rule')
            == 'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing_none': policy.get('postprocessing') == 'none',
        'tta_none': policy.get('test_time_augmentation') == 'none',
        'both_locked_splits_required': (
            policy.get('locked_evaluation_splits') == ['val', 'test']
        ),
        'sources_initially_stable': all(
            locked_sources_unchanged(source_records).values()
        ),
    }
    locked_checks_or_raise(
        upstream_checks, 'Frozen deployment-policy authorization failed'
    )

    manifest = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    manifest['subject_id'] = manifest['subject_id'].astype(str)
    locked_rows = manifest[
        manifest['dev_split'].astype(str).isin(['val', 'test'])
    ].copy().sort_values(['dev_split', 'subject_id'])
    split_counts = locked_rows['dev_split'].value_counts().to_dict()
    manifest_checks = {
        'locked_subject_count_is_46': len(locked_rows) == 46,
        'locked_subjects_unique': locked_rows['subject_id'].is_unique,
        'split_counts_exact': split_counts == EXPECTED_LOCKED_SPLITS,
        **locked_manifest_input_contract_checks(locked_rows),
    }
    locked_checks_or_raise(manifest_checks, 'Locked manifest audit failed')

    prediction_contract = {
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'deployment_policy_sha256': policy_hash,
        'checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_folds': list(checkpoint_folds),
        'weight_state': 'ema_model_state',
        'ensemble_rule': 'arithmetic_mean_of_five_fold_ema_probabilities',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'subject_ids': locked_rows['subject_id'].tolist(),
        'target_access_before_prediction_commit': False,
    }
    prediction_contract_hash = locked_canonical_hash(prediction_contract)
    LOCKED_PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)
    LOCKED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    if LOCKED_PREDICTION_GATE_PATH.exists():
        prediction_gate = load_json(LOCKED_PREDICTION_GATE_PATH)
        gate_checks = {
            'prediction_complete': prediction_gate.get('prediction_complete') is True,
            'contract_matches': (
                prediction_gate.get('prediction_contract_sha256')
                == prediction_contract_hash
            ),
            'all_46_subjects': prediction_gate.get('subjects_predicted') == 46,
            'locked_targets_not_read': (
                prediction_gate.get('locked_validation_targets_read') is False
                and prediction_gate.get('test_targets_read') is False
            ),
            'prediction_archives_stable': all(
                Path(record['path']).is_file()
                and file_sha256(record['path']) == record['sha256']
                for record in prediction_gate.get('prediction_manifest', [])
            ),
        }
        locked_checks_or_raise(
            gate_checks, 'Existing target-blind prediction gate rejected'
        )
        prediction_manifest = prediction_gate['prediction_manifest']
        print('\nVerified complete target-blind prediction gate; skipping inference.')
    else:
        if not torch.cuda.is_available():
            raise RuntimeError('Final five-model evaluation requires a CUDA GPU.')
        device = torch.device('cuda')
        set_determinism(PILOT_SEED + 9_000)
        gc.collect()
        torch.cuda.empty_cache()
        models = []
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL TARGET-BLIND FIVE-MODEL INFERENCE STARTING')
        print('=' * 80)
        print('Locked targets read:                False')
        print('Optimizer constructed / steps:      False / 0')
        for record in checkpoints:
            fold = int(record['development_fold'])
            checkpoint = torch.load(
                record['best_checkpoint_path'], map_location='cpu',
                weights_only=False,
            )
            checkpoint_checks = {
                'fold_matches': checkpoint.get('development_fold') == fold,
                'best_epoch_matches': int(checkpoint.get('epoch', -1))
                == int(record['best_epoch']),
                'parameter_count_matches': int(
                    checkpoint.get('parameter_count', -1)
                ) == EXPECTED_V22_PARAMETERS,
                'training_series_matches': (
                    checkpoint.get('replication_series_version')
                    == REPLICATION_SERIES_VERSION
                ),
                'training_revision_matches': (
                    checkpoint.get('code_revision') == PILOT_CODE_REVISION
                ),
                'checkpoint_policy_is_strict_numeric': (
                    checkpoint.get('checkpoint_selection_policy')
                    == 'strict_numerical_maximum'
                ),
                'ema_state_present': isinstance(
                    checkpoint.get('ema_model_state'), dict
                ),
            }
            locked_checks_or_raise(
                checkpoint_checks, f'Fold-{fold} checkpoint identity failed'
            )
            model = VascuFateNetV22(pilot_schema)
            if sum(parameter.numel() for parameter in model.parameters()) != (
                EXPECTED_V22_PARAMETERS
            ):
                raise RuntimeError(f'Fold-{fold}: model parameter count changed.')
            model.load_state_dict(checkpoint['ema_model_state'], strict=True)
            model = model.to(device).eval()
            for parameter in model.parameters():
                parameter.requires_grad_(False)
            if not all(
                bool(torch.isfinite(parameter).all().item())
                for parameter in model.parameters()
            ):
                raise RuntimeError(f'Fold-{fold}: EMA parameters are non-finite.')
            models.append((fold, model))
            del checkpoint
            print(f'Loaded fold-{fold} strict-best EMA checkpoint.')

        prior_index = (
            load_json(LOCKED_PREDICTION_INDEX_PATH)
            if LOCKED_PREDICTION_INDEX_PATH.exists() else {}
        )
        if prior_index and prior_index.get('prediction_contract_sha256') != (
            prediction_contract_hash
        ):
            raise RuntimeError(
                'Partial prediction index belongs to another contract; '
                'refusing to overwrite it.'
            )
        prediction_manifest = []
        for index, row in enumerate(locked_rows.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            split = str(row['dev_split'])
            archive_path = locked_prediction_path(subject_id)
            existing_archive = locked_validate_prediction_archive(
                archive_path, prediction_contract_hash, subject_id
            )
            if existing_archive is not None:
                prediction_manifest.append(existing_archive)
                print(
                    f'Target-blind prediction {index:02d}/46: {subject_id} '
                    '(verified resume)'
                )
                continue
            local_sum = None
            proposal_sum = None
            fold_masks = []
            fold_presence = []
            fold_routed_volume = []
            for fold, model in models:
                global RUNTIME_FOLD
                RUNTIME_FOLD = fold
                result = locked_infer_case(model, row, device)
                local_probability = result['probability']
                proposal_probability = result['proposal_probability']
                if local_sum is None:
                    local_sum = np.zeros_like(local_probability, dtype=np.float32)
                    proposal_sum = np.zeros_like(
                        proposal_probability, dtype=np.float32
                    )
                local_sum += local_probability
                proposal_sum += proposal_probability
                fold_masks.append(local_probability >= local_threshold)
                fold_presence.append(result['presence_probability'])
                fold_routed_volume.append(result['routed_volume_ml'])
                del result, local_probability, proposal_probability
            ensemble_local_probability = local_sum / 5.0
            ensemble_proposal_probability = proposal_sum / 5.0
            ensemble_local = ensemble_local_probability >= local_threshold
            ensemble_proposal = (
                ensemble_proposal_probability >= proposal_threshold
            )
            shape = tuple(int(v) for v in ensemble_local.shape)
            fold_packed = np.stack([
                locked_pack_mask(mask) for mask in fold_masks
            ], axis=0)
            local_probability_hash = hashlib.sha256(
                np.ascontiguousarray(
                    ensemble_local_probability.astype('<f4')
                ).tobytes()
            ).hexdigest()
            proposal_probability_hash = hashlib.sha256(
                np.ascontiguousarray(
                    ensemble_proposal_probability.astype('<f4')
                ).tobytes()
            ).hexdigest()
            locked_atomic_npz(
                archive_path,
                subject_id=np.asarray(subject_id, dtype=np.str_),
                dev_split=np.asarray(split, dtype=np.str_),
                native_shape=np.asarray(shape, dtype=np.int64),
                contract_sha256=np.asarray(
                    prediction_contract_hash, dtype=np.str_
                ),
                fold_ids=np.asarray(EXPECTED_ENSEMBLE_FOLDS, dtype=np.int64),
                fold_local_masks_packed=fold_packed,
                ensemble_local_mask_packed=locked_pack_mask(ensemble_local),
                ensemble_proposal_mask_packed=locked_pack_mask(
                    ensemble_proposal
                ),
                fold_predicted_voxels=np.asarray([
                    int(mask.sum()) for mask in fold_masks
                ], dtype=np.int64),
                fold_presence_probabilities=np.asarray(
                    fold_presence, dtype=np.float32
                ),
                fold_routed_volume_ml=np.asarray(
                    fold_routed_volume, dtype=np.float32
                ),
                ensemble_local_probability_sha256=np.asarray(
                    local_probability_hash, dtype=np.str_
                ),
                ensemble_proposal_probability_sha256=np.asarray(
                    proposal_probability_hash, dtype=np.str_
                ),
            )
            manifest_row = locked_prediction_manifest_row(
                subject_id, split, archive_path, shape, fold_masks,
                ensemble_local, ensemble_proposal,
                ensemble_local_probability, ensemble_proposal_probability,
                fold_presence, fold_routed_volume,
            )
            prediction_manifest.append(manifest_row)
            atomic_write_json({
                'prediction_contract_sha256': prediction_contract_hash,
                'completed_subjects': len(prediction_manifest),
                'prediction_manifest': prediction_manifest,
                'locked_validation_targets_read': False,
                'test_targets_read': False,
            }, LOCKED_PREDICTION_INDEX_PATH)
            print(
                f'Target-blind prediction {index:02d}/46: {subject_id} '
                f'({split})'
            )
            del (
                local_sum, proposal_sum, fold_masks,
                ensemble_local_probability, ensemble_proposal_probability,
                ensemble_local, ensemble_proposal,
            )
            gc.collect()
            torch.cuda.empty_cache()

        
        normalized_manifest = []
        for row in locked_rows.to_dict('records'):
            subject_id = str(row['subject_id'])
            archive_path = locked_prediction_path(subject_id)
            locked_validate_prediction_archive(
                archive_path, prediction_contract_hash, subject_id
            )
            with np.load(archive_path, allow_pickle=False) as data:
                shape = tuple(int(v) for v in data['native_shape'].tolist())
                fold_masks = [
                    locked_unpack_mask(bits, shape)
                    for bits in data['fold_local_masks_packed']
                ]
                ensemble_local = locked_unpack_mask(
                    data['ensemble_local_mask_packed'], shape
                )
                ensemble_proposal = locked_unpack_mask(
                    data['ensemble_proposal_mask_packed'], shape
                )
                fold_presence = np.asarray(
                    data['fold_presence_probabilities'], dtype=np.float32
                )
                fold_routed = np.asarray(
                    data['fold_routed_volume_ml'], dtype=np.float32
                )
                probability_stats = {
                    'local_mean': float('nan'), 'local_max': float('nan'),
                    'proposal_mean': float('nan'), 'proposal_max': float('nan'),
                }
            pairwise = locked_pairwise_dice(fold_masks)
            vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
            union_voxels = int((vote_count > 0).sum())
            disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
            normalized_manifest.append({
                'subject_id': subject_id,
                'dev_split': str(row['dev_split']),
                'path': str(archive_path),
                'sha256': file_sha256(archive_path),
                'native_shape': list(shape),
                'ensemble_local_voxels': int(ensemble_local.sum()),
                'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
                'fold_local_predicted_voxels': [
                    int(mask.sum()) for mask in fold_masks
                ],
                'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
                'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
                'disagreement_union_fraction': float(
                    disagreement / max(union_voxels, 1)
                ),
                'mean_presence_probability': float(fold_presence.mean()),
                'mean_routed_volume_ml': float(fold_routed.mean()),
            })
        prediction_manifest = normalized_manifest
        prediction_gate = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'evaluation_version': LOCKED_EVALUATION_VERSION,
            'code_revision': LOCKED_EVALUATION_CODE_REVISION,
            'prediction_complete': True,
            'prediction_contract': prediction_contract,
            'prediction_contract_sha256': prediction_contract_hash,
            'subjects_predicted': len(prediction_manifest),
            'prediction_manifest': prediction_manifest,
            'ensemble_members': 5,
            'model_constructed_for_inference_only': True,
            'optimizer_constructed': False,
            'optimizer_steps': 0,
            'training_performed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'targets_opened_only_after_this_gate': True,
            'locked_manifest_checks': manifest_checks,
        }
        prediction_checks = {
            'all_46_predictions_present': len(prediction_manifest) == 46,
            'subjects_unique': len({
                item['subject_id'] for item in prediction_manifest
            }) == 46,
            'archives_stable': all(
                Path(item['path']).is_file()
                and file_sha256(item['path']) == item['sha256']
                for item in prediction_manifest
            ),
            'sources_still_unchanged': all(
                locked_sources_unchanged(source_records).values()
            ),
            'locked_manifest_checks_all_true': bool(
                manifest_checks and all(manifest_checks.values())
            ),
        }
        locked_checks_or_raise(
            prediction_checks, 'Target-blind prediction commit failed'
        )
        prediction_gate['checks'] = prediction_checks
        atomic_write_json(prediction_gate, LOCKED_PREDICTION_GATE_PATH)
        del models
        gc.collect()
        torch.cuda.empty_cache()

    
    
    print('\nTarget-blind prediction gate committed. Loading both locked target sets.')
    target_cache = {}
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
        target, soft, spacing = locked_load_target(row, shape)
        target_cache[subject_id] = {
            'target': target,
            'soft_nonzero_voxels': int((soft > 0).sum()),
            'spacing': spacing,
        }
    if len(target_cache) != 46:
        raise RuntimeError('Both locked target sets were not loaded completely.')

    case_rows = []
    component_rows = []
    member_rows = []
    diversity_rows = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
            fold_masks = [
                locked_unpack_mask(bits, shape)
                for bits in data['fold_local_masks_packed']
            ]
            ensemble_local = locked_unpack_mask(
                data['ensemble_local_mask_packed'], shape
            )
            ensemble_proposal = locked_unpack_mask(
                data['ensemble_proposal_mask_packed'], shape
            )
            fold_presence = np.asarray(
                data['fold_presence_probabilities'], dtype=np.float32
            )
            fold_routed = np.asarray(
                data['fold_routed_volume_ml'], dtype=np.float32
            )
        target = target_cache[subject_id]['target']
        spacing = target_cache[subject_id]['spacing']
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
        local_metrics = locked_mask_metrics(
            ensemble_local, target, spacing, 'local'
        )
        proposal_metrics = locked_mask_metrics(
            ensemble_proposal, target, spacing, 'proposal'
        )
        case_record = {
            'subject_id': subject_id,
            'dev_split': split,
            'lesion_volume_bin': lesion_bin,
            'hard_metric_eligible': bool(hard_eligible),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': local_metrics['target_voxels'],
            'predicted_voxels': local_metrics['predicted_voxels'],
            'intersection_voxels': local_metrics['intersection_voxels'],
            'dice': local_metrics['dice'],
            'voxel_precision': local_metrics['voxel_precision'],
            'voxel_recall': local_metrics['voxel_recall'],
            'target_volume_ml': local_metrics['target_voxels'] * voxel_volume_ml,
            'predicted_volume_ml': (
                local_metrics['predicted_voxels'] * voxel_volume_ml
            ),
            'lesion_true_positive': local_metrics['lesion_true_positive'],
            'lesion_false_positive': local_metrics['lesion_false_positive'],
            'lesion_false_negative': local_metrics['lesion_false_negative'],
            'component_true_positive': local_metrics['component_true_positive'],
            'component_total': local_metrics['component_total'],
            'small_component_true_positive': (
                local_metrics['small_component_true_positive']
            ),
            'small_component_total': local_metrics['small_component_total'],
            'proposal_predicted_voxels': proposal_metrics['predicted_voxels'],
            'proposal_dice': proposal_metrics['dice'],
            'proposal_component_true_positive': (
                proposal_metrics['component_true_positive']
            ),
            'proposal_component_total': proposal_metrics['component_total'],
            'mean_presence_probability': float(fold_presence.mean()),
            'mean_routed_volume_ml': float(fold_routed.mean()),
            'soft_nonzero_voxels': target_cache[subject_id][
                'soft_nonzero_voxels'
            ],
            'prediction_archive_path': str(archive_path),
            'prediction_archive_sha256': file_sha256(archive_path),
        }
        for fold, mask in zip(EXPECTED_ENSEMBLE_FOLDS, fold_masks):
            fold_metrics = locked_mask_metrics(mask, target, spacing, 'local')
            case_record[f'fold_{fold}_dice'] = fold_metrics['dice']
            member_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'development_fold_model': fold,
                'hard_metric_eligible': bool(hard_eligible),
                'lesion_volume_bin': lesion_bin,
                'target_voxels': fold_metrics['target_voxels'],
                'predicted_voxels': fold_metrics['predicted_voxels'],
                'intersection_voxels': fold_metrics['intersection_voxels'],
                'dice': fold_metrics['dice'],
                'target_volume_ml': fold_metrics['target_voxels'] * voxel_volume_ml,
                'predicted_volume_ml': (
                    fold_metrics['predicted_voxels'] * voxel_volume_ml
                ),
                'lesion_true_positive': fold_metrics['lesion_true_positive'],
                'lesion_false_positive': fold_metrics['lesion_false_positive'],
                'lesion_false_negative': fold_metrics['lesion_false_negative'],
                'component_true_positive': fold_metrics['component_true_positive'],
                'component_total': fold_metrics['component_total'],
                'small_component_true_positive': (
                    fold_metrics['small_component_true_positive']
                ),
                'small_component_total': fold_metrics['small_component_total'],
            })
        pairwise = locked_pairwise_dice(fold_masks)
        vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
        union = int((vote_count > 0).sum())
        disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
        diversity_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
            'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
            'maximum_pairwise_fold_mask_dice': float(np.max(pairwise)),
            'prediction_union_voxels': union,
            'prediction_disagreement_voxels': disagreement,
            'disagreement_union_fraction': disagreement / max(union, 1),
            'ensemble_dice': local_metrics['dice'],
        })
        for component in local_metrics['components']:
            proposal_component = next(
                item for item in proposal_metrics['components']
                if int(item['component_id']) == int(component['component_id'])
            )
            component_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'component_id': int(component['component_id']),
                'component_tier': component['component_tier'],
                'component_voxels': int(component['component_voxels']),
                'component_volume_ml': float(component['component_volume_ml']),
                'local_maximum_iou': float(component['maximum_component_iou']),
                'local_detected_iou_ge_0p10': bool(
                    float(component['maximum_component_iou']) >= 0.10
                ),
                'proposal_maximum_iou': float(
                    proposal_component['maximum_component_iou']
                ),
                'proposal_detected_iou_ge_0p05': bool(
                    float(proposal_component['maximum_component_iou']) >= 0.05
                ),
            })
        case_rows.append(case_record)

    case_df = pd.DataFrame(case_rows).sort_values(['dev_split', 'subject_id'])
    component_df = pd.DataFrame(component_rows)
    member_case_df = pd.DataFrame(member_rows)
    diversity_df = pd.DataFrame(diversity_rows)
    split_summary_rows = []
    member_summary_rows = []
    bin_rows = []
    bootstrap_rows = []
    split_summaries = {}
    for split in ['val', 'test']:
        split_frame = case_df[case_df['dev_split'] == split].copy()
        summary = locked_aggregate_case_rows(split_frame)
        summary['dev_split'] = split
        split_summaries[split] = summary
        split_summary_rows.append(summary)
        bootstrap_rows.extend(locked_bootstrap_split(
            split_frame, split, LOCKED_BOOTSTRAP_REPLICATES,
            LOCKED_BOOTSTRAP_SEED + (0 if split == 'val' else 1),
        ))
        for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
            bin_frame = split_frame[
                split_frame['lesion_volume_bin'] == lesion_bin
            ].copy()
            if bin_frame.empty:
                continue
            eligible_bin = bin_frame[
                bin_frame['hard_metric_eligible'].astype(bool)
            ]
            positive_bin = eligible_bin[
                eligible_bin['target_voxels'].astype(int) > 0
            ]
            bin_rows.append({
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'subjects': int(len(bin_frame)),
                'positive_subjects': int(len(positive_bin)),
                'mean_dice': float(positive_bin['dice'].mean())
                if len(positive_bin) else float('nan'),
                'median_dice': float(positive_bin['dice'].median())
                if len(positive_bin) else float('nan'),
                'pooled_global_dice': float(
                    2.0 * positive_bin['intersection_voxels'].sum()
                    / max(
                        positive_bin['predicted_voxels'].sum()
                        + positive_bin['target_voxels'].sum(), 1,
                    )
                ) if len(positive_bin) else float('nan'),
                'mean_target_volume_ml': float(
                    positive_bin['target_volume_ml'].mean()
                ) if len(positive_bin) else 0.0,
                'mean_predicted_volume_ml': float(
                    eligible_bin['predicted_volume_ml'].mean()
                ) if len(eligible_bin) else 0.0,
            })
        for fold in EXPECTED_ENSEMBLE_FOLDS:
            member_frame = member_case_df[
                (member_case_df['dev_split'] == split)
                & (member_case_df['development_fold_model'] == fold)
            ].copy()
            member_summary = locked_aggregate_case_rows(member_frame)
            member_summary.update({
                'dev_split': split,
                'model': f'fold_{fold}_ema',
                'development_fold_model': fold,
            })
            member_summary_rows.append(member_summary)
        ensemble_member_summary = summary.copy()
        ensemble_member_summary.update({
            'model': 'fivefold_probability_ensemble',
            'development_fold_model': -1,
        })
        member_summary_rows.append(ensemble_member_summary)

    split_summary_df = pd.DataFrame(split_summary_rows)
    bin_df = pd.DataFrame(bin_rows)
    member_summary_df = pd.DataFrame(member_summary_rows)
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    prediction_audit_df = pd.DataFrame(prediction_manifest)
    source_manifest_df = pd.DataFrame(source_records)
    oof_reference = policy['primary_oof_metrics']
    generalization_rows = []
    metric_mapping = {
        'positive_mean_dice': 'positive_case_mean_dice',
        'positive_median_dice': 'positive_case_median_dice',
        'positive_global_dice': 'pooled_global_dice',
        'lesion_f1': 'lesion_f1',
        'tiny_small_subject_mean_dice': (
            'tiny_small_subject_weighted_mean_dice'
        ),
        'component_recall': 'component_recall',
        'small_component_recall': 'small_component_recall',
    }
    for split in ['val', 'test']:
        for locked_metric, oof_metric in metric_mapping.items():
            locked_value = float(split_summaries[split][locked_metric])
            oof_value = float(oof_reference[oof_metric])
            generalization_rows.append({
                'dev_split': split,
                'metric': locked_metric,
                'development_oof_value': oof_value,
                'locked_value': locked_value,
                'absolute_change_from_oof': locked_value - oof_value,
                'relative_change_from_oof': (
                    (locked_value - oof_value) / abs(oof_value)
                    if abs(oof_value) > 1e-12 else float('nan')
                ),
                'interpretation': (
                    'confirmatory_transfer_diagnostic_not_a_tuning_signal'
                ),
            })
    generalization_df = pd.DataFrame(generalization_rows)
    ensemble_gain = {}
    for split in ['val', 'test']:
        members = member_summary_df[
            (member_summary_df['dev_split'] == split)
            & (member_summary_df['development_fold_model'] >= 0)
        ]
        ensemble_gain[split] = {
            'mean_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_mean_dice']
                - members['positive_mean_dice'].mean()
            ),
            'global_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_global_dice']
                - members['positive_global_dice'].mean()
            ),
            'lesion_f1_gain_over_mean_member': float(
                split_summaries[split]['lesion_f1']
                - members['lesion_f1'].mean()
            ),
        }

    final_checks = {
        **{
            f'locked_manifest_{name}': bool(passed)
            for name, passed in manifest_checks.items()
        },
        'prediction_gate_complete': load_json(
            LOCKED_PREDICTION_GATE_PATH
        ).get('prediction_complete') is True,
        'prediction_contract_matches': load_json(
            LOCKED_PREDICTION_GATE_PATH
        ).get('prediction_contract_sha256') == prediction_contract_hash,
        'all_46_cases_evaluated': len(case_df) == 46,
        'validation_has_23_cases': int(
            (case_df['dev_split'] == 'val').sum()
        ) == 23,
        'test_has_23_cases': int(
            (case_df['dev_split'] == 'test').sum()
        ) == 23,
        'all_subjects_unique': case_df['subject_id'].is_unique,
        'all_case_metrics_finite': bool(np.isfinite(case_df[[
            'dice', 'voxel_precision', 'voxel_recall',
            'target_volume_ml', 'predicted_volume_ml',
        ]].to_numpy(dtype=float)).all()),
        'both_split_summaries_finite': bool(np.isfinite(
            split_summary_df.select_dtypes(include=[np.number]).to_numpy()
        ).all()),
        'bootstrap_complete': len(bootstrap_df) == 20,
        'bootstrap_intervals_finite': bool(np.isfinite(bootstrap_df[[
            'point_estimate', 'ci_95_lower', 'ci_95_upper',
            'bootstrap_standard_error',
        ]].to_numpy(dtype=float)).all()),
        'five_members_plus_ensemble_per_split': len(member_summary_df) == 12,
        'generalization_rows_complete': len(generalization_df) == 14,
        'generalization_core_values_finite': bool(np.isfinite(
            generalization_df[[
                'development_oof_value', 'locked_value',
                'absolute_change_from_oof',
            ]].to_numpy(dtype=float)
        ).all()),
        'frozen_local_threshold_unchanged': local_threshold == 0.40,
        'frozen_proposal_threshold_unchanged': proposal_threshold == 0.35,
        'postprocessing_remained_none': policy['postprocessing'] == 'none',
        'tta_remained_none': policy['test_time_augmentation'] == 'none',
        'architecture_unchanged': True,
        'checkpoints_unchanged': all(
            locked_sources_unchanged(source_records).values()
        ),
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'post_freeze_tuning_not_performed': True,
        'validation_and_test_evaluated_in_one_frozen_execution': True,
    }
    locked_checks_or_raise(final_checks, 'Final locked evaluation failed')

    atomic_write_csv(source_manifest_df, LOCKED_SOURCE_MANIFEST_PATH)
    atomic_write_csv(prediction_audit_df, LOCKED_PREDICTION_AUDIT_PATH)
    atomic_write_csv(case_df, LOCKED_CASE_METRICS_PATH)
    atomic_write_csv(component_df, LOCKED_COMPONENT_METRICS_PATH)
    atomic_write_csv(split_summary_df, LOCKED_SPLIT_SUMMARY_PATH)
    atomic_write_csv(bin_df, LOCKED_BIN_METRICS_PATH)
    atomic_write_csv(member_summary_df, LOCKED_MEMBER_METRICS_PATH)
    atomic_write_csv(diversity_df, LOCKED_DIVERSITY_PATH)
    atomic_write_csv(bootstrap_df, LOCKED_BOOTSTRAP_PATH)
    atomic_write_csv(generalization_df, LOCKED_GENERALIZATION_PATH)
    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'selected_architecture': policy['selected_architecture'],
        'ensemble_members': 5,
        'checkpoint_weight_state': 'ema_model_state',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'development_oof_reference': oof_reference,
        'ensemble_gain_over_mean_member': ensemble_gain,
        'bootstrap_replicates': LOCKED_BOOTSTRAP_REPLICATES,
        'metric_scope': (
            'hard-metric-eligible positive cases for Dice; hard-metric-eligible '
            'cases for lesion detection; thresholds frozen from development OOF'
        ),
        'validation_used_for_tuning': False,
        'test_used_for_tuning': False,
        'post_freeze_tuning_performed': False,
        'scientific_interpretation': (
            'Validation and test are confirmatory locked evaluations. The '
            'development OOF estimate remains separately reported and is not '
            'pooled with either locked split as a primary performance estimate.'
        ),
    }
    atomic_write_json(report, LOCKED_REPORT_PATH)
    output_paths = {
        'target_blind_prediction_gate': LOCKED_PREDICTION_GATE_PATH,
        'source_artifact_manifest': LOCKED_SOURCE_MANIFEST_PATH,
        'prediction_audit': LOCKED_PREDICTION_AUDIT_PATH,
        'case_metrics': LOCKED_CASE_METRICS_PATH,
        'component_metrics': LOCKED_COMPONENT_METRICS_PATH,
        'split_summary': LOCKED_SPLIT_SUMMARY_PATH,
        'lesion_bin_metrics': LOCKED_BIN_METRICS_PATH,
        'ensemble_member_metrics': LOCKED_MEMBER_METRICS_PATH,
        'ensemble_diversity_metrics': LOCKED_DIVERSITY_PATH,
        'bootstrap_intervals': LOCKED_BOOTSTRAP_PATH,
        'oof_to_locked_generalization': LOCKED_GENERALIZATION_PATH,
        'evaluation_report': LOCKED_REPORT_PATH,
    }
    output_manifest = {
        name: {'path': str(path), 'sha256': file_sha256(path)}
        for name, path in output_paths.items()
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'evaluation_complete': True,
        'evaluated_splits': ['val', 'test'],
        'subjects_evaluated': {'val': 23, 'test': 23},
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'ensemble_members': 5,
        'ensemble_checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_weight_state': 'ema_model_state',
        'ensemble_probability_rule': (
            'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'prediction_contract_sha256': prediction_contract_hash,
        'prediction_gate_path': str(LOCKED_PREDICTION_GATE_PATH),
        'prediction_gate_sha256': file_sha256(LOCKED_PREDICTION_GATE_PATH),
        'source_artifact_manifest': source_records,
        'output_artifact_manifest': output_manifest,
        'final_checks': final_checks,
        'locked_validation_targets_read': True,
        'test_targets_read': True,
        'targets_read_only_after_target_blind_prediction_commit': True,
        'model_constructed_for_inference_only': True,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'post_freeze_tuning_performed': False,
        'further_model_or_threshold_tuning_authorized': False,
        'next_action': (
            'Report the frozen OOF, locked validation, and locked test results; '
            'perform no further model selection or threshold tuning.'
        ),
    }
    atomic_write_json(gate, LOCKED_FINAL_GATE_PATH)

    print('\nLocked split summary:')
    display(split_summary_df[[
        'dev_split', 'subjects', 'positive_subjects',
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]])
    print('\nLocked lesion-volume-bin metrics:')
    display(bin_df)
    print('\nLocked 95% descriptive bootstrap intervals:')
    display(bootstrap_df)
    print('\nEnsemble versus member models:')
    display(member_summary_df[[
        'dev_split', 'model', 'positive_mean_dice',
        'positive_global_dice', 'lesion_f1',
        'tiny_small_subject_mean_dice',
    ]])
    print('\nDevelopment OOF to locked-split generalization:')
    display(generalization_df[[
        'dev_split', 'metric', 'development_oof_value', 'locked_value',
        'absolute_change_from_oof',
    ]])
    print('\n' + '=' * 80)
    print('TRIAL8 FINAL LOCKED VALIDATION/TEST ENSEMBLE EVALUATION PASSED')
    print('=' * 80)
    for split in ['val', 'test']:
        item = split_summaries[split]
        print(
            f'{split.upper():10s} mean / median / global Dice: '
            f'{item["positive_mean_dice"]:.4f} / '
            f'{item["positive_median_dice"]:.4f} / '
            f'{item["positive_global_dice"]:.4f}'
        )
        print(
            f'{split.upper():10s} lesion F1 / component recall: '
            f'{item["lesion_f1"]:.4f} / '
            f'{item["component_recall"]:.4f}'
        )
    print('Frozen local / proposal threshold:  0.40 / 0.35')
    print('Postprocessing / TTA:               none / none')
    print('Post-freeze tuning performed:       False')
    print('Further tuning authorized:          False')
    print(f'Final report:                       {LOCKED_REPORT_PATH}')
    print(f'Final gate:                         {LOCKED_FINAL_GATE_PATH}')
    print('\nNext: report results and limitations; do not tune on these splits.')


def locked_optional_text(value):
    if value is None:
        return ''
    try:
        if bool(pd.isna(value)):
            return ''
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return '' if text.lower() in {'nan', 'none', 'null'} else text


def locked_optional_bool(value):
    if value is None:
        return None
    try:
        if bool(pd.isna(value)):
            return None
    except (TypeError, ValueError):
        pass
    try:
        return bool(as_bool(value))
    except ValueError:
        return None


def locked_artifact_manifest_is_stable(manifest):
    if not isinstance(manifest, dict) or not manifest:
        return False
    return all(
        isinstance(record, dict)
        and Path(str(record.get('path', ''))).is_file()
        and file_sha256(Path(str(record['path']))) == record.get('sha256')
        for record in manifest.values()
    )


def locked_record_manifest_is_stable(records):
    if not isinstance(records, list) or not records:
        return False
    return all(
        isinstance(record, dict)
        and Path(str(record.get('path', ''))).is_file()
        and file_sha256(Path(str(record['path']))) == record.get('sha256')
        for record in records
    )


def run_locked_metric_reconciliation():
    existing = locked_existing_final_is_valid()
    if existing is not None:
        summaries = existing['split_summaries']
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL LOCKED METRIC RECONCILIATION ALREADY COMPLETE')
        print('=' * 80)
        for split in ['val', 'test']:
            item = summaries[split]
            print(
                f'{split.upper():10s} mean/global Dice:             '
                f'{item["positive_mean_dice"]:.4f} / '
                f'{item["positive_global_dice"]:.4f}'
            )
        print(f'Final gate: {LOCKED_FINAL_GATE_PATH}')
        print('No checkpoint, prediction, target, or result artifact was rewritten.')
        return

    required_paths = [
        OOF_DEPLOYMENT_POLICY_PATH, OOF_AGGREGATION_GATE_PATH,
        MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
        MULTIMODAL_GATE_PATH, LOCKED_PREDICTION_GATE_PATH,
        SOURCE_FLAWED_FINAL_GATE_PATH,
    ]
    missing = [str(path) for path in required_paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(
            'Metric-reconciliation prerequisites are missing:\n'
            + '\n'.join(missing)
        )

    policy = load_json(OOF_DEPLOYMENT_POLICY_PATH)
    oof_gate = load_json(OOF_AGGREGATION_GATE_PATH)
    source_prediction_gate = load_json(LOCKED_PREDICTION_GATE_PATH)
    source_flawed_gate = load_json(SOURCE_FLAWED_FINAL_GATE_PATH)
    policy_hash = file_sha256(OOF_DEPLOYMENT_POLICY_PATH)
    source_prediction_gate_hash = file_sha256(LOCKED_PREDICTION_GATE_PATH)
    source_flawed_gate_hash = file_sha256(SOURCE_FLAWED_FINAL_GATE_PATH)

    source_records = [
        locked_source_record(OOF_DEPLOYMENT_POLICY_PATH, 'deployment_policy'),
        locked_source_record(OOF_AGGREGATION_GATE_PATH, 'oof_freeze_gate'),
        locked_source_record(MULTIMODAL_MANIFEST_PATH, 'multimodal_manifest'),
        locked_source_record(FOLD_MANIFEST_PATH, 'fold_manifest'),
        locked_source_record(GRAPH_SCHEMA_PATH, 'graph_schema'),
        locked_source_record(MULTIMODAL_GATE_PATH, 'multimodal_gate'),
    ]
    checkpoints = sorted(
        policy.get('ensemble_checkpoints', []),
        key=lambda item: int(item['development_fold']),
    )
    for record in checkpoints:
        fold = int(record['development_fold'])
        source_records.append(locked_source_record(
            record['best_checkpoint_path'], 'best_ema_checkpoint', fold,
        ))
        graph_scaler = Path(resolve_gate_mapping(
            multimodal_gate['graph_scaler_paths'], fold
        ))
        clinical_scaler = Path(resolve_gate_mapping(
            multimodal_gate['clinical_scaler_paths'], fold
        ))
        source_records.append(locked_source_record(
            graph_scaler, 'fold_graph_scaler', fold
        ))
        source_records.append(locked_source_record(
            clinical_scaler, 'fold_clinical_scaler', fold
        ))
    base_source_count = len(source_records)
    source_records.extend([
        locked_source_record(
            LOCKED_PREDICTION_GATE_PATH, 'source_r2_prediction_gate'
        ),
        locked_source_record(
            SOURCE_FLAWED_FINAL_GATE_PATH,
            'source_r2_superseded_metric_gate',
        ),
    ])

    local_threshold = float(policy.get('frozen_local_threshold', float('nan')))
    proposal_threshold = float(
        policy.get('frozen_proposal_threshold', float('nan'))
    )
    checkpoint_folds = tuple(
        int(record['development_fold']) for record in checkpoints
    )
    upstream_checks = {
        'oof_aggregation_complete': oof_gate.get('aggregation_complete') is True,
        'locked_evaluation_authorized': (
            oof_gate.get('locked_evaluation_authorized') is True
            and policy.get('locked_evaluation_authorized') is True
        ),
        'aggregation_version_matches': (
            oof_gate.get('aggregation_version')
            == EXPECTED_OOF_AGGREGATION_VERSION
            and policy.get('aggregation_version')
            == EXPECTED_OOF_AGGREGATION_VERSION
        ),
        'aggregation_code_revision_matches': (
            oof_gate.get('code_revision') == EXPECTED_OOF_CODE_REVISION
            and policy.get('code_revision') == EXPECTED_OOF_CODE_REVISION
        ),
        'policy_hash_matches_oof_output_manifest': (
            oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('path') == str(OOF_DEPLOYMENT_POLICY_PATH)
            and oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('sha256') == policy_hash
        ),
        'oof_final_checks_all_true': bool(
            oof_gate.get('final_checks')
            and all(oof_gate.get('final_checks', {}).values())
        ),
        'five_checkpoints_exact': checkpoint_folds == EXPECTED_ENSEMBLE_FOLDS,
        'ema_weight_state_exact': all(
            record.get('weight_state') == 'ema_model_state'
            for record in checkpoints
        ) and policy.get('checkpoint_weight_state') == 'ema_model_state',
        'checkpoint_hashes_match_policy': all(
            file_sha256(record['best_checkpoint_path'])
            == record['best_checkpoint_sha256']
            for record in checkpoints
        ),
        'checkpoint_parameters_match': all(
            int(record['parameter_count']) == EXPECTED_V22_PARAMETERS
            for record in checkpoints
        ),
        'local_threshold_exact': math.isclose(
            local_threshold, EXPECTED_FROZEN_LOCAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_exact': math.isclose(
            proposal_threshold, EXPECTED_FROZEN_PROPOSAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'probability_ensemble_rule_exact': (
            policy.get('ensemble_probability_rule')
            == 'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing_none': policy.get('postprocessing') == 'none',
        'tta_none': policy.get('test_time_augmentation') == 'none',
        'both_locked_splits_required': (
            policy.get('locked_evaluation_splits') == ['val', 'test']
        ),
        'base_source_manifest_has_21_records': base_source_count == 21,
        'source_r2_output_artifacts_stable': locked_artifact_manifest_is_stable(
            source_flawed_gate.get('output_artifact_manifest', {})
        ),
        'source_r2_upstream_sources_stable': locked_record_manifest_is_stable(
            source_flawed_gate.get('source_artifact_manifest', [])
        ),
        'sources_initially_stable': all(
            locked_sources_unchanged(source_records).values()
        ),
    }
    locked_checks_or_raise(
        upstream_checks, 'Frozen reconciliation authorization failed'
    )

    manifest = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    manifest['subject_id'] = manifest['subject_id'].astype(str)
    locked_rows = manifest[
        manifest['dev_split'].astype(str).isin(['val', 'test'])
    ].copy().sort_values(['dev_split', 'subject_id']).reset_index(drop=True)
    split_counts = locked_rows['dev_split'].value_counts().to_dict()
    manifest_checks = {
        'locked_subject_count_is_46': len(locked_rows) == 46,
        'locked_subjects_unique': locked_rows['subject_id'].is_unique,
        'split_counts_exact': split_counts == EXPECTED_LOCKED_SPLITS,
        **locked_manifest_input_contract_checks(locked_rows),
    }
    locked_checks_or_raise(manifest_checks, 'Locked manifest audit failed')

    source_prediction_contract = {
        'evaluation_version': SOURCE_PREDICTION_VERSION,
        'code_revision': SOURCE_PREDICTION_CODE_REVISION,
        'deployment_policy_sha256': policy_hash,
        'checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_folds': list(checkpoint_folds),
        'weight_state': 'ema_model_state',
        'ensemble_rule': 'arithmetic_mean_of_five_fold_ema_probabilities',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'subject_ids': locked_rows['subject_id'].tolist(),
        'target_access_before_prediction_commit': False,
    }
    source_prediction_contract_hash = locked_canonical_hash(
        source_prediction_contract
    )
    source_gate_manifest = source_prediction_gate.get('prediction_manifest', [])
    source_gate_subjects = [
        str(record.get('subject_id', '')) for record in source_gate_manifest
    ]
    source_gate_records = {
        str(record.get('subject_id', '')): record
        for record in source_gate_manifest
    }
    source_r2_checks = {
        'prediction_gate_version_matches': (
            source_prediction_gate.get('evaluation_version')
            == SOURCE_PREDICTION_VERSION
        ),
        'prediction_gate_revision_matches': (
            source_prediction_gate.get('code_revision')
            == SOURCE_PREDICTION_CODE_REVISION
        ),
        'prediction_complete': (
            source_prediction_gate.get('prediction_complete') is True
        ),
        'prediction_contract_hash_matches': (
            source_prediction_gate.get('prediction_contract_sha256')
            == source_prediction_contract_hash
        ),
        'stored_prediction_contract_hashes_exactly': (
            locked_canonical_hash(
                source_prediction_gate.get('prediction_contract', {})
            ) == source_prediction_contract_hash
        ),
        'all_46_subjects_predicted': (
            int(source_prediction_gate.get('subjects_predicted', -1)) == 46
        ),
        'prediction_manifest_has_46_records': len(source_gate_manifest) == 46,
        'prediction_manifest_subjects_unique': (
            len(set(source_gate_subjects)) == 46
        ),
        'prediction_manifest_subject_set_exact': (
            set(source_gate_subjects) == set(locked_rows['subject_id'])
        ),
        'prediction_gate_checks_all_true': bool(
            source_prediction_gate.get('checks')
            and all(source_prediction_gate.get('checks', {}).values())
        ),
        'targets_were_sealed_during_prediction': (
            source_prediction_gate.get('locked_validation_targets_read') is False
            and source_prediction_gate.get('test_targets_read') is False
        ),
        'source_final_gate_version_matches': (
            source_flawed_gate.get('evaluation_version')
            == SOURCE_PREDICTION_VERSION
        ),
        'source_final_gate_revision_matches': (
            source_flawed_gate.get('code_revision')
            == SOURCE_PREDICTION_CODE_REVISION
        ),
        'source_final_gate_complete': (
            source_flawed_gate.get('evaluation_complete') is True
        ),
        'source_final_checks_all_true': bool(
            source_flawed_gate.get('final_checks')
            and all(source_flawed_gate.get('final_checks', {}).values())
        ),
        'source_final_prediction_contract_matches': (
            source_flawed_gate.get('prediction_contract_sha256')
            == source_prediction_contract_hash
        ),
        'source_failure_signature_test_positive_count_zero': (
            int(source_flawed_gate.get('split_summaries', {}).get(
                'test', {}
            ).get('positive_subjects', -1)) == 0
        ),
        'source_validation_has_positive_cases': (
            int(source_flawed_gate.get('split_summaries', {}).get(
                'val', {}
            ).get('positive_subjects', 0)) > 0
        ),
        'source_prediction_gate_hash_unchanged': (
            file_sha256(LOCKED_PREDICTION_GATE_PATH)
            == source_prediction_gate_hash
        ),
        'source_flawed_gate_hash_unchanged': (
            file_sha256(SOURCE_FLAWED_FINAL_GATE_PATH)
            == source_flawed_gate_hash
        ),
    }
    locked_checks_or_raise(
        source_r2_checks, 'Frozen R2 prediction source rejected'
    )

    prediction_manifest = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        gate_record = source_gate_records[subject_id]
        gate_path_text = (
            gate_record.get('path') or gate_record.get('prediction_path') or ''
        )
        validated = locked_validate_prediction_archive(
            archive_path, source_prediction_contract_hash, subject_id
        )
        if validated is None:
            raise RuntimeError(
                f'{subject_id}: frozen R2 prediction archive is missing.'
            )
        with np.load(archive_path, allow_pickle=False) as data:
            archive_split = scalar_string(data['dev_split'])
            shape = tuple(int(v) for v in data['native_shape'].tolist())
            fold_ids = tuple(int(v) for v in data['fold_ids'].tolist())
            fold_masks = [
                locked_unpack_mask(bits, shape)
                for bits in data['fold_local_masks_packed']
            ]
            ensemble_local = locked_unpack_mask(
                data['ensemble_local_mask_packed'], shape
            )
            ensemble_proposal = locked_unpack_mask(
                data['ensemble_proposal_mask_packed'], shape
            )
            fold_voxels = np.asarray(
                data['fold_predicted_voxels'], dtype=np.int64
            )
            fold_presence = np.asarray(
                data['fold_presence_probabilities'], dtype=np.float64
            )
            fold_routed = np.asarray(
                data['fold_routed_volume_ml'], dtype=np.float64
            )
            local_probability_hash = scalar_string(
                data['ensemble_local_probability_sha256']
            )
            proposal_probability_hash = scalar_string(
                data['ensemble_proposal_probability_sha256']
            )
        gate_archive_hash = (
            gate_record.get('sha256')
            or gate_record.get('prediction_sha256')
        )
        archive_checks = {
            'manifest_path_matches_frozen_archive': (
                Path(gate_path_text) == archive_path
            ),
            'manifest_hash_matches_archive': (
                gate_archive_hash == file_sha256(archive_path)
            ),
            'archive_split_matches_manifest': archive_split == split,
            'fold_ids_exact': fold_ids == EXPECTED_ENSEMBLE_FOLDS,
            'five_fold_masks': len(fold_masks) == 5,
            'fold_voxel_counts_shape_exact': fold_voxels.shape == (5,),
            'fold_voxel_counts_match_masks': bool(np.array_equal(
                fold_voxels,
                np.asarray([int(mask.sum()) for mask in fold_masks]),
            )),
            'presence_shape_exact': fold_presence.shape == (5,),
            'routed_volume_shape_exact': fold_routed.shape == (5,),
            'presence_values_finite': bool(np.isfinite(fold_presence).all()),
            'routed_volume_values_finite_nonnegative': bool(
                np.isfinite(fold_routed).all() and (fold_routed >= 0).all()
            ),
            'probability_hashes_well_formed': (
                len(local_probability_hash) == 64
                and len(proposal_probability_hash) == 64
            ),
        }
        locked_checks_or_raise(
            archive_checks, f'{subject_id}: frozen R2 archive audit failed'
        )
        pairwise = locked_pairwise_dice(fold_masks)
        vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
        union_voxels = int((vote_count > 0).sum())
        disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
        prediction_manifest.append({
            'subject_id': subject_id,
            'dev_split': split,
            'path': str(archive_path),
            'sha256': file_sha256(archive_path),
            'native_shape': list(shape),
            'ensemble_local_voxels': int(ensemble_local.sum()),
            'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
            'fold_local_predicted_voxels': [
                int(mask.sum()) for mask in fold_masks
            ],
            'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
            'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
            'disagreement_union_fraction': float(
                disagreement / max(union_voxels, 1)
            ),
            'mean_presence_probability': float(fold_presence.mean()),
            'mean_routed_volume_ml': float(fold_routed.mean()),
            'source_prediction_contract_sha256': (
                source_prediction_contract_hash
            ),
        })
        source_records.append(locked_source_record(
            archive_path, 'source_r2_prediction_archive'
        ))

    prediction_checks = {
        'all_46_archives_verified': len(prediction_manifest) == 46,
        'archive_subjects_unique': len({
            item['subject_id'] for item in prediction_manifest
        }) == 46,
        'archive_subject_set_exact': {
            item['subject_id'] for item in prediction_manifest
        } == set(locked_rows['subject_id']),
        'source_manifest_has_69_records': len(source_records) == 69,
        'all_source_records_stable_before_target_access': all(
            locked_sources_unchanged(source_records).values()
        ),
        'source_and_reconciliation_roots_are_distinct': (
            SOURCE_PREDICTION_ROOT.resolve() != LOCKED_OUTPUT_ROOT.resolve()
        ),
    }
    locked_checks_or_raise(
        prediction_checks, 'Frozen prediction archive commit verification failed'
    )

    LOCKED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print('\n' + '=' * 80)
    print('TRIAL8 R3 READ-ONLY LOCKED METRIC RECONCILIATION STARTING')
    print('=' * 80)
    print('Frozen R2 prediction archives:       46 / 46 verified')
    print('Inference repeated:                  False')
    print('Model / optimizer constructed:       False / False')
    print('Predictions or checkpoints rewritten: False')
    print('Thresholds changed:                  False')
    print('Loading both locked target sets only after source-gate verification.')

    target_cache = {}
    policy_audit_rows = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
        target, soft, spacing = locked_load_target(row, shape)
        derived = locked_derive_target_policy(target, soft, spacing)
        manifest_hard = locked_optional_bool(
            row.get('trial8_hard_metric_eligible')
        )
        manifest_bin = locked_optional_text(
            row.get('trial8_effective_lesion_volume_bin')
        )
        manifest_mode = locked_optional_text(
            row.get('trial8_supervision_mode')
        )
        hard_match = (
            manifest_hard is not None
            and manifest_hard == derived['hard_metric_eligible']
        )
        bin_match = manifest_bin == derived['lesion_volume_bin']
        mode_match = manifest_mode == derived['supervision_mode']
        target_cache[subject_id] = {
            'target': target,
            'soft_nonzero_voxels': derived['soft_support_voxels'],
            'spacing': spacing,
            'policy': derived,
        }
        policy_audit_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'derived_supervision_mode': derived['supervision_mode'],
            'derived_hard_metric_eligible': (
                derived['hard_metric_eligible']
            ),
            'derived_lesion_volume_bin': derived['lesion_volume_bin'],
            'hard_voxels': derived['hard_voxels'],
            'soft_support_voxels': derived['soft_support_voxels'],
            'hard_target_volume_ml': derived['hard_target_volume_ml'],
            'soft_target_volume_ml': derived['soft_target_volume_ml'],
            'effective_target_volume_ml': (
                derived['effective_target_volume_ml']
            ),
            'manifest_supervision_mode': manifest_mode,
            'manifest_hard_metric_eligible': manifest_hard,
            'manifest_lesion_volume_bin': manifest_bin,
            'supervision_mode_matches_manifest': bool(mode_match),
            'hard_metric_eligibility_matches_manifest': bool(hard_match),
            'lesion_volume_bin_matches_manifest': bool(bin_match),
            'manifest_policy_used_for_scoring': False,
            'target_policy_derived_after_prediction_commit': True,
        })
    if len(target_cache) != 46:
        raise RuntimeError('Both locked target sets were not loaded completely.')
    target_policy_df = pd.DataFrame(policy_audit_rows).sort_values(
        ['dev_split', 'subject_id']
    )
    validation_policy = target_policy_df[
        target_policy_df['dev_split'] == 'val'
    ]
    test_policy = target_policy_df[target_policy_df['dev_split'] == 'test']
    test_hard_positive_count = int((
        test_policy['derived_hard_metric_eligible'].astype(bool)
        & (test_policy['hard_voxels'].astype(int) > 0)
    ).sum())
    target_policy_checks = {
        'all_46_target_policies_derived': len(target_policy_df) == 46,
        'validation_has_23_derived_policies': len(validation_policy) == 23,
        'test_has_23_derived_policies': len(test_policy) == 23,
        'validation_declared_supervision_modes_reproduced': bool(
            validation_policy.loc[
                validation_policy['manifest_supervision_mode'].astype(str) != '',
                'supervision_mode_matches_manifest',
            ].all()
        ),
        'validation_hard_eligibility_reproduced_exactly': bool(
            validation_policy[
                'hard_metric_eligibility_matches_manifest'
            ].all()
        ),
        'validation_lesion_bins_reproduced_exactly': bool(
            validation_policy['lesion_volume_bin_matches_manifest'].all()
        ),
        'test_contains_target_derived_hard_positive_cases': (
            test_hard_positive_count > 0
        ),
        'manifest_policy_not_used_for_scoring': not bool(
            target_policy_df['manifest_policy_used_for_scoring'].any()
        ),
    }
    locked_checks_or_raise(
        target_policy_checks,
        'Target-derived policy reconciliation failed before metric calculation',
    )

    case_rows = []
    component_rows = []
    member_rows = []
    diversity_rows = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
            fold_masks = [
                locked_unpack_mask(bits, shape)
                for bits in data['fold_local_masks_packed']
            ]
            ensemble_local = locked_unpack_mask(
                data['ensemble_local_mask_packed'], shape
            )
            ensemble_proposal = locked_unpack_mask(
                data['ensemble_proposal_mask_packed'], shape
            )
            fold_presence = np.asarray(
                data['fold_presence_probabilities'], dtype=np.float32
            )
            fold_routed = np.asarray(
                data['fold_routed_volume_ml'], dtype=np.float32
            )
        target = target_cache[subject_id]['target']
        spacing = target_cache[subject_id]['spacing']
        derived = target_cache[subject_id]['policy']
        voxel_volume_ml = derived['voxel_volume_ml']
        hard_eligible = derived['hard_metric_eligible']
        lesion_bin = derived['lesion_volume_bin']
        local_metrics = locked_mask_metrics(
            ensemble_local, target, spacing, 'local'
        )
        proposal_metrics = locked_mask_metrics(
            ensemble_proposal, target, spacing, 'proposal'
        )
        case_record = {
            'subject_id': subject_id,
            'dev_split': split,
            'supervision_mode': derived['supervision_mode'],
            'lesion_volume_bin': lesion_bin,
            'hard_metric_eligible': bool(hard_eligible),
            'metric_policy_source': 'target_derived_after_prediction_commit',
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': local_metrics['target_voxels'],
            'predicted_voxels': local_metrics['predicted_voxels'],
            'intersection_voxels': local_metrics['intersection_voxels'],
            'dice': local_metrics['dice'],
            'voxel_precision': local_metrics['voxel_precision'],
            'voxel_recall': local_metrics['voxel_recall'],
            'target_volume_ml': local_metrics['target_voxels'] * voxel_volume_ml,
            'predicted_volume_ml': (
                local_metrics['predicted_voxels'] * voxel_volume_ml
            ),
            'lesion_true_positive': local_metrics['lesion_true_positive'],
            'lesion_false_positive': local_metrics['lesion_false_positive'],
            'lesion_false_negative': local_metrics['lesion_false_negative'],
            'component_true_positive': local_metrics['component_true_positive'],
            'component_total': local_metrics['component_total'],
            'small_component_true_positive': (
                local_metrics['small_component_true_positive']
            ),
            'small_component_total': local_metrics['small_component_total'],
            'proposal_predicted_voxels': proposal_metrics['predicted_voxels'],
            'proposal_dice': proposal_metrics['dice'],
            'proposal_component_true_positive': (
                proposal_metrics['component_true_positive']
            ),
            'proposal_component_total': proposal_metrics['component_total'],
            'mean_presence_probability': float(fold_presence.mean()),
            'mean_routed_volume_ml': float(fold_routed.mean()),
            'soft_nonzero_voxels': derived['soft_support_voxels'],
            'prediction_archive_path': str(archive_path),
            'prediction_archive_sha256': file_sha256(archive_path),
        }
        for fold, mask in zip(EXPECTED_ENSEMBLE_FOLDS, fold_masks):
            fold_metrics = locked_mask_metrics(mask, target, spacing, 'local')
            case_record[f'fold_{fold}_dice'] = fold_metrics['dice']
            member_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'development_fold_model': fold,
                'hard_metric_eligible': bool(hard_eligible),
                'lesion_volume_bin': lesion_bin,
                'target_voxels': fold_metrics['target_voxels'],
                'predicted_voxels': fold_metrics['predicted_voxels'],
                'intersection_voxels': fold_metrics['intersection_voxels'],
                'dice': fold_metrics['dice'],
                'target_volume_ml': (
                    fold_metrics['target_voxels'] * voxel_volume_ml
                ),
                'predicted_volume_ml': (
                    fold_metrics['predicted_voxels'] * voxel_volume_ml
                ),
                'lesion_true_positive': fold_metrics['lesion_true_positive'],
                'lesion_false_positive': fold_metrics['lesion_false_positive'],
                'lesion_false_negative': fold_metrics['lesion_false_negative'],
                'component_true_positive': (
                    fold_metrics['component_true_positive']
                ),
                'component_total': fold_metrics['component_total'],
                'small_component_true_positive': (
                    fold_metrics['small_component_true_positive']
                ),
                'small_component_total': fold_metrics['small_component_total'],
            })
        pairwise = locked_pairwise_dice(fold_masks)
        vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
        union = int((vote_count > 0).sum())
        disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
        diversity_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
            'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
            'maximum_pairwise_fold_mask_dice': float(np.max(pairwise)),
            'prediction_union_voxels': union,
            'prediction_disagreement_voxels': disagreement,
            'disagreement_union_fraction': disagreement / max(union, 1),
            'ensemble_dice': local_metrics['dice'],
        })
        for component in local_metrics['components']:
            proposal_component = next(
                item for item in proposal_metrics['components']
                if int(item['component_id']) == int(component['component_id'])
            )
            component_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'component_id': int(component['component_id']),
                'component_tier': component['component_tier'],
                'component_voxels': int(component['component_voxels']),
                'component_volume_ml': float(component['component_volume_ml']),
                'local_maximum_iou': float(component['maximum_component_iou']),
                'local_detected_iou_ge_0p10': bool(
                    float(component['maximum_component_iou']) >= 0.10
                ),
                'proposal_maximum_iou': float(
                    proposal_component['maximum_component_iou']
                ),
                'proposal_detected_iou_ge_0p05': bool(
                    float(proposal_component['maximum_component_iou']) >= 0.05
                ),
            })
        case_rows.append(case_record)

    case_df = pd.DataFrame(case_rows).sort_values(['dev_split', 'subject_id'])
    component_df = pd.DataFrame(component_rows)
    member_case_df = pd.DataFrame(member_rows)
    diversity_df = pd.DataFrame(diversity_rows)
    split_summary_rows = []
    member_summary_rows = []
    bin_rows = []
    bootstrap_rows = []
    split_summaries = {}
    for split in ['val', 'test']:
        split_frame = case_df[case_df['dev_split'] == split].copy()
        summary = locked_aggregate_case_rows(split_frame)
        summary['dev_split'] = split
        split_summaries[split] = summary
        split_summary_rows.append(summary)
        bootstrap_rows.extend(locked_bootstrap_split(
            split_frame, split, LOCKED_BOOTSTRAP_REPLICATES,
            LOCKED_BOOTSTRAP_SEED + (0 if split == 'val' else 1),
        ))
        for lesion_bin in [
            'empty', 'tiny', 'small', 'medium', 'large', 'very_large'
        ]:
            bin_frame = split_frame[
                split_frame['lesion_volume_bin'] == lesion_bin
            ].copy()
            if bin_frame.empty:
                continue
            eligible_bin = bin_frame[
                bin_frame['hard_metric_eligible'].astype(bool)
            ]
            positive_bin = eligible_bin[
                eligible_bin['target_voxels'].astype(int) > 0
            ]
            bin_rows.append({
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'subjects': int(len(bin_frame)),
                'positive_subjects': int(len(positive_bin)),
                'mean_dice': float(positive_bin['dice'].mean())
                if len(positive_bin) else float('nan'),
                'median_dice': float(positive_bin['dice'].median())
                if len(positive_bin) else float('nan'),
                'pooled_global_dice': float(
                    2.0 * positive_bin['intersection_voxels'].sum()
                    / max(
                        positive_bin['predicted_voxels'].sum()
                        + positive_bin['target_voxels'].sum(), 1,
                    )
                ) if len(positive_bin) else float('nan'),
                'mean_target_volume_ml': float(
                    positive_bin['target_volume_ml'].mean()
                ) if len(positive_bin) else 0.0,
                'mean_predicted_volume_ml': float(
                    eligible_bin['predicted_volume_ml'].mean()
                ) if len(eligible_bin) else 0.0,
            })
        for fold in EXPECTED_ENSEMBLE_FOLDS:
            member_frame = member_case_df[
                (member_case_df['dev_split'] == split)
                & (member_case_df['development_fold_model'] == fold)
            ].copy()
            member_summary = locked_aggregate_case_rows(member_frame)
            member_summary.update({
                'dev_split': split,
                'model': f'fold_{fold}_ema',
                'development_fold_model': fold,
            })
            member_summary_rows.append(member_summary)
        ensemble_member_summary = summary.copy()
        ensemble_member_summary.update({
            'model': 'fivefold_probability_ensemble',
            'development_fold_model': -1,
        })
        member_summary_rows.append(ensemble_member_summary)

    split_summary_df = pd.DataFrame(split_summary_rows)
    bin_df = pd.DataFrame(bin_rows)
    member_summary_df = pd.DataFrame(member_summary_rows)
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    prediction_audit_df = pd.DataFrame(prediction_manifest)
    source_manifest_df = pd.DataFrame(source_records)
    oof_reference = policy['primary_oof_metrics']
    generalization_rows = []
    metric_mapping = {
        'positive_mean_dice': 'positive_case_mean_dice',
        'positive_median_dice': 'positive_case_median_dice',
        'positive_global_dice': 'pooled_global_dice',
        'lesion_f1': 'lesion_f1',
        'tiny_small_subject_mean_dice': (
            'tiny_small_subject_weighted_mean_dice'
        ),
        'component_recall': 'component_recall',
        'small_component_recall': 'small_component_recall',
    }
    for split in ['val', 'test']:
        for locked_metric, oof_metric in metric_mapping.items():
            locked_value = float(split_summaries[split][locked_metric])
            oof_value = float(oof_reference[oof_metric])
            generalization_rows.append({
                'dev_split': split,
                'metric': locked_metric,
                'development_oof_value': oof_value,
                'locked_value': locked_value,
                'absolute_change_from_oof': locked_value - oof_value,
                'relative_change_from_oof': (
                    (locked_value - oof_value) / abs(oof_value)
                    if abs(oof_value) > 1e-12 else float('nan')
                ),
                'interpretation': (
                    'confirmatory_transfer_diagnostic_not_a_tuning_signal'
                ),
            })
    generalization_df = pd.DataFrame(generalization_rows)
    ensemble_gain = {}
    for split in ['val', 'test']:
        members = member_summary_df[
            (member_summary_df['dev_split'] == split)
            & (member_summary_df['development_fold_model'] >= 0)
        ]
        ensemble_gain[split] = {
            'mean_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_mean_dice']
                - members['positive_mean_dice'].mean()
            ),
            'global_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_global_dice']
                - members['positive_global_dice'].mean()
            ),
            'lesion_f1_gain_over_mean_member': float(
                split_summaries[split]['lesion_f1']
                - members['lesion_f1'].mean()
            ),
        }

    source_validation_summary = source_flawed_gate['split_summaries']['val']
    validation_float_metrics = [
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]
    validation_integer_metrics = [
        'subjects', 'hard_metric_eligible_subjects', 'positive_subjects',
        'intersection_voxels', 'predicted_voxels_positive_cases',
        'target_voxels_positive_cases', 'lesion_true_positive',
        'lesion_false_positive', 'lesion_false_negative',
        'component_true_positive', 'component_total',
        'small_component_true_positive', 'small_component_total',
    ]
    validation_reproduction_checks = {
        f'validation_{name}_reproduced': math.isclose(
            float(split_summaries['val'][name]),
            float(source_validation_summary[name]),
            rel_tol=0.0, abs_tol=1e-12,
        )
        for name in validation_float_metrics
    }
    validation_reproduction_checks.update({
        f'validation_{name}_reproduced': (
            int(split_summaries['val'][name])
            == int(source_validation_summary[name])
        )
        for name in validation_integer_metrics
    })

    final_checks = {
        **{
            f'upstream_{name}': bool(passed)
            for name, passed in upstream_checks.items()
        },
        **{
            f'locked_manifest_{name}': bool(passed)
            for name, passed in manifest_checks.items()
        },
        **{
            f'source_r2_{name}': bool(passed)
            for name, passed in source_r2_checks.items()
        },
        **{
            f'prediction_{name}': bool(passed)
            for name, passed in prediction_checks.items()
        },
        **{
            f'target_policy_{name}': bool(passed)
            for name, passed in target_policy_checks.items()
        },
        **validation_reproduction_checks,
        'all_46_cases_evaluated': len(case_df) == 46,
        'validation_has_23_cases': int(
            (case_df['dev_split'] == 'val').sum()
        ) == 23,
        'test_has_23_cases': int(
            (case_df['dev_split'] == 'test').sum()
        ) == 23,
        'test_positive_count_comes_from_targets': (
            int(split_summaries['test']['positive_subjects'])
            == test_hard_positive_count
            and test_hard_positive_count > 0
        ),
        'all_subjects_unique': case_df['subject_id'].is_unique,
        'all_case_metrics_finite': bool(np.isfinite(case_df[[
            'dice', 'voxel_precision', 'voxel_recall',
            'target_volume_ml', 'predicted_volume_ml',
        ]].to_numpy(dtype=float)).all()),
        'both_split_summaries_finite': bool(np.isfinite(
            split_summary_df.select_dtypes(include=[np.number]).to_numpy()
        ).all()),
        'bootstrap_complete': len(bootstrap_df) == 20,
        'bootstrap_intervals_finite': bool(np.isfinite(bootstrap_df[[
            'point_estimate', 'ci_95_lower', 'ci_95_upper',
            'bootstrap_standard_error',
        ]].to_numpy(dtype=float)).all()),
        'five_members_plus_ensemble_per_split': len(member_summary_df) == 12,
        'generalization_rows_complete': len(generalization_df) == 14,
        'generalization_core_values_finite': bool(np.isfinite(
            generalization_df[[
                'development_oof_value', 'locked_value',
                'absolute_change_from_oof',
            ]].to_numpy(dtype=float)
        ).all()),
        'source_prediction_contract_unchanged': (
            file_sha256(LOCKED_PREDICTION_GATE_PATH)
            == source_prediction_gate_hash
        ),
        'source_flawed_gate_unchanged': (
            file_sha256(SOURCE_FLAWED_FINAL_GATE_PATH)
            == source_flawed_gate_hash
        ),
        'all_source_predictions_and_checkpoints_unchanged': all(
            locked_sources_unchanged(source_records).values()
        ),
        'frozen_local_threshold_unchanged': local_threshold == 0.40,
        'frozen_proposal_threshold_unchanged': proposal_threshold == 0.35,
        'postprocessing_remained_none': policy['postprocessing'] == 'none',
        'tta_remained_none': policy['test_time_augmentation'] == 'none',
        'architecture_unchanged': True,
        'model_not_constructed': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'inference_not_repeated': True,
        'training_not_performed': True,
        'post_freeze_tuning_not_performed': True,
        'validation_and_test_reconciled_in_one_frozen_execution': True,
    }
    locked_checks_or_raise(final_checks, 'Final locked reconciliation failed')

    atomic_write_csv(source_manifest_df, LOCKED_SOURCE_MANIFEST_PATH)
    atomic_write_csv(prediction_audit_df, LOCKED_PREDICTION_AUDIT_PATH)
    atomic_write_csv(target_policy_df, LOCKED_TARGET_POLICY_AUDIT_PATH)
    atomic_write_csv(case_df, LOCKED_CASE_METRICS_PATH)
    atomic_write_csv(component_df, LOCKED_COMPONENT_METRICS_PATH)
    atomic_write_csv(split_summary_df, LOCKED_SPLIT_SUMMARY_PATH)
    atomic_write_csv(bin_df, LOCKED_BIN_METRICS_PATH)
    atomic_write_csv(member_summary_df, LOCKED_MEMBER_METRICS_PATH)
    atomic_write_csv(diversity_df, LOCKED_DIVERSITY_PATH)
    atomic_write_csv(bootstrap_df, LOCKED_BOOTSTRAP_PATH)
    atomic_write_csv(generalization_df, LOCKED_GENERALIZATION_PATH)
    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'source_prediction_version': SOURCE_PREDICTION_VERSION,
        'source_prediction_code_revision': SOURCE_PREDICTION_CODE_REVISION,
        'source_prediction_contract_sha256': (
            source_prediction_contract_hash
        ),
        'selected_architecture': policy['selected_architecture'],
        'ensemble_members': 5,
        'checkpoint_weight_state': 'ema_model_state',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'development_oof_reference': oof_reference,
        'ensemble_gain_over_mean_member': ensemble_gain,
        'bootstrap_replicates': LOCKED_BOOTSTRAP_REPLICATES,
        'metric_policy': {
            'source': 'target_derived_after_prediction_commit',
            'hard_and_soft': 'hard_metric_eligible',
            'soft_only_subvoxel': 'excluded_from_hard_metrics',
            'true_empty': 'eligible_for_lesion_detection_metrics',
            'volume_bins_ml': {
                'tiny': '[0,1)', 'small': '[1,5)',
                'medium': '[5,25)', 'large': '[25,75)',
                'very_large': '[75,infinity)',
            },
            'validation_policy_reproduced_exactly': True,
            'test_manifest_policy_used_for_scoring': False,
        },
        'metric_scope': (
            'hard-metric-eligible positive cases for Dice; hard-metric-eligible '
            'cases for lesion detection; thresholds frozen from development OOF'
        ),
        'source_r2_test_metrics_status': (
            'superseded_due_to_sealed_manifest_metric_eligibility'
        ),
        'source_predictions_reused': True,
        'inference_repeated': False,
        'validation_used_for_tuning': False,
        'test_used_for_tuning': False,
        'post_freeze_tuning_performed': False,
        'scientific_interpretation': (
            'Validation and test are confirmatory locked evaluations. R3 '
            'changes only post-commit metric eligibility accounting and '
            'reproduces the complete R2 validation metrics exactly. The '
            'development OOF estimate remains separately reported.'
        ),
    }
    atomic_write_json(report, LOCKED_REPORT_PATH)
    output_paths = {
        'target_blind_prediction_gate': LOCKED_PREDICTION_GATE_PATH,
        'source_artifact_manifest': LOCKED_SOURCE_MANIFEST_PATH,
        'prediction_audit': LOCKED_PREDICTION_AUDIT_PATH,
        'target_policy_audit': LOCKED_TARGET_POLICY_AUDIT_PATH,
        'case_metrics': LOCKED_CASE_METRICS_PATH,
        'component_metrics': LOCKED_COMPONENT_METRICS_PATH,
        'split_summary': LOCKED_SPLIT_SUMMARY_PATH,
        'lesion_bin_metrics': LOCKED_BIN_METRICS_PATH,
        'ensemble_member_metrics': LOCKED_MEMBER_METRICS_PATH,
        'ensemble_diversity_metrics': LOCKED_DIVERSITY_PATH,
        'bootstrap_intervals': LOCKED_BOOTSTRAP_PATH,
        'oof_to_locked_generalization': LOCKED_GENERALIZATION_PATH,
        'evaluation_report': LOCKED_REPORT_PATH,
    }
    output_manifest = {
        name: {'path': str(path), 'sha256': file_sha256(path)}
        for name, path in output_paths.items()
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'evaluation_complete': True,
        'evaluated_splits': ['val', 'test'],
        'subjects_evaluated': {'val': 23, 'test': 23},
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'ensemble_members': 5,
        'ensemble_checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_weight_state': 'ema_model_state',
        'ensemble_probability_rule': (
            'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'prediction_contract_sha256': source_prediction_contract_hash,
        'prediction_gate_path': str(LOCKED_PREDICTION_GATE_PATH),
        'prediction_gate_sha256': source_prediction_gate_hash,
        'source_superseded_metric_gate_path': str(
            SOURCE_FLAWED_FINAL_GATE_PATH
        ),
        'source_superseded_metric_gate_sha256': source_flawed_gate_hash,
        'source_artifact_manifest': source_records,
        'output_artifact_manifest': output_manifest,
        'final_checks': final_checks,
        'locked_validation_targets_read': True,
        'test_targets_read': True,
        'targets_read_only_after_target_blind_prediction_commit': True,
        'target_policy_derived_after_prediction_commit': True,
        'validation_policy_reproduced_exactly': True,
        'test_manifest_policy_used_for_scoring': False,
        'source_predictions_reused': True,
        'inference_repeated': False,
        'model_constructed_for_metric_reconciliation': False,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'post_freeze_tuning_performed': False,
        'source_r2_test_metrics_superseded': True,
        'further_model_or_threshold_tuning_authorized': False,
        'next_action': (
            'Report the frozen OOF and reconciled locked validation/test '
            'results; perform no further model selection or threshold tuning.'
        ),
    }
    atomic_write_json(gate, LOCKED_FINAL_GATE_PATH)

    print('\nTarget-derived policy audit by split:')
    display(target_policy_df.groupby('dev_split', as_index=False).agg(
        subjects=('subject_id', 'count'),
        hard_metric_eligible=('derived_hard_metric_eligible', 'sum'),
        hard_positive_cases=('hard_voxels', lambda values: int((values > 0).sum())),
        supervision_mode_matches=('supervision_mode_matches_manifest', 'sum'),
        hard_eligibility_matches=(
            'hard_metric_eligibility_matches_manifest', 'sum'
        ),
        lesion_bin_matches=('lesion_volume_bin_matches_manifest', 'sum'),
    ))
    print('\nReconciled locked split summary:')
    display(split_summary_df[[
        'dev_split', 'subjects', 'positive_subjects',
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]])
    print('\nLocked lesion-volume-bin metrics:')
    display(bin_df)
    print('\nLocked 95% descriptive bootstrap intervals:')
    display(bootstrap_df)
    print('\nEnsemble versus member models:')
    display(member_summary_df[[
        'dev_split', 'model', 'positive_mean_dice',
        'positive_global_dice', 'lesion_f1',
        'tiny_small_subject_mean_dice',
    ]])
    print('\nDevelopment OOF to locked-split generalization:')
    display(generalization_df[[
        'dev_split', 'metric', 'development_oof_value', 'locked_value',
        'absolute_change_from_oof',
    ]])
    print('\n' + '=' * 80)
    print('TRIAL8 FINAL LOCKED METRIC RECONCILIATION PASSED')
    print('=' * 80)
    print('Frozen R2 predictions reused:        True')
    print('Inference / training repeated:       False / False')
    print('Validation policy reproduced:        True')
    print('R2 all-zero test metrics superseded: True')
    for split in ['val', 'test']:
        item = split_summaries[split]
        print(
            f'{split.upper():10s} mean / median / global Dice: '
            f'{item["positive_mean_dice"]:.4f} / '
            f'{item["positive_median_dice"]:.4f} / '
            f'{item["positive_global_dice"]:.4f}'
        )
        print(
            f'{split.upper():10s} lesion F1 / component recall: '
            f'{item["lesion_f1"]:.4f} / '
            f'{item["component_recall"]:.4f}'
        )
    print('Frozen local / proposal threshold:  0.40 / 0.35')
    print('Postprocessing / TTA:               none / none')
    print('Post-freeze tuning performed:       False')
    print('Further tuning authorized:          False')
    print(f'Final report:                       {LOCKED_REPORT_PATH}')
    print(f'Final gate:                         {LOCKED_FINAL_GATE_PATH}')
    print('\nNext: report results and limitations; do not tune on these splits.')


run_locked_metric_reconciliation()


In [ ]:
import sys
import os
import json
import math
import hashlib
import subprocess
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path


def ensure_package(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name
        ])
        return __import__(import_name)


np = ensure_package('numpy')
pd = ensure_package('pandas')
nib = ensure_package('nibabel')
torch = ensure_package('torch')

import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
from IPython.display import display

drive.mount('/mnt/storage')






PROJECT_ROOT = Path('/path/to/project')
R2_VERSION = 'v2_vascufate_ras_0p8x0p8x2_r2'
MODEL_VERSION = 'trial8_vascufate_net_v2_2_global_local_arterial'
BLOCK1_VERSION = 'v1_component_proposal_arterial_projection_contract'
PREFLIGHT_VERSION = 'v2_2_global_local_model_construction_r2'
CODE_REVISION = 'r2_context_dependent_diagnostic_gate_activation'

EXPECTED_SUBJECTS = 148
EXPECTED_SPLITS = {'train': 102, 'val': 23, 'test': 23}
EXPECTED_FOLD0_FIT = 82
EXPECTED_FOLD0_HELDOUT = 20
EXPECTED_V2_PARAMETERS = 15_428_468
MAXIMUM_NEW_PARAMETERS = 2_500_000
MAXIMUM_TOTAL_PARAMETERS = EXPECTED_V2_PARAMETERS + MAXIMUM_NEW_PARAMETERS
EXPECTED_TARGET_POLICY_MANIFEST_SHA256 = (
    'cc4b87af8452145dccf0f9a3c74da7b6aad0701d7bb72eb5246f3b9021be8ce5'
)
EXPECTED_FOLD_MANIFEST_SHA256 = (
    '2424e41b43502d50db89d4c3d1e90bf64a7cce6266ec99825ba865f2297d70a9'
)
EXPECTED_MULTIMODAL_MANIFEST_SHA256 = (
    '6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235'
)
EXPECTED_GRAPH_SCHEMA_SHA256 = (
    'e55de076307d822ea57bc2f235b1854329144e422bfe1585f553f7a1f78c0211'
)

FOLD0 = 0



RUNTIME_FOLD = FOLD0
SEED = 20261001
LOCAL_PATCH_SIZE = (128, 128, 64)
GLOBAL_CANVAS_SHAPE = (288, 320, 128)
GLOBAL_INPUT_SHAPE = (144, 160, 64)
GLOBAL_DOWNSAMPLE = (2, 2, 2)
NUM_TERRITORY_NODES = 30
NUM_VESSEL_NODES = 13
GRAPH_HIDDEN = 64
GRAPH_HEADS = 2
GRAPH_LAYERS = 2
RELATION_BASES = 3
GRAPH_DROPOUT = 0.10
ATLAS_DROPOUT_PROBABILITY = 0.20
FILM_BOUND = 0.10
FUSION_BOUND = 0.10
CLINICAL_INPUT_DIM = 44
CLINICAL_EMBED_DIM = 32
OCCLUSION_VALUE_DIM = 7
OCCLUSION_EMBED_DIM = 32
VASCULAR_CHANNELS = (8, 16, 32, 48)
VASCULAR_INPUT_CHANNELS = 5
CONFIDENCE_DIM = 4
GLOBAL_CHANNELS = (12, 24, 40, 64, 96)
GLOBAL_CONTEXT_DIM = 64
ARTERIAL_HIDDEN = 64
GPU_MEMORY_LIMIT_GB = 20.0

EXPECTED_INPUT_CHANNELS = [
    'ncct_brain_window', 'ncct_broad_window', 'cta', 'cbf', 'cbv',
    'mtt', 'tmax', 'relative_cbf', 'relative_cbv', 'delta_cbf',
    'delta_cbv', 'delta_mtt', 'delta_tmax', 'perfusion_valid',
    'contralateral_pair_valid',
]
SOURCE_VESSEL_FEATURE_NAMES = [
    'present', 'volume_ml', 'relative_centroid_x_mm',
    'relative_centroid_y_mm', 'relative_centroid_z_mm', 'extent_x_mm',
    'extent_y_mm', 'extent_z_mm', 'cta_mean', 'cta_p90', 'cbf_mean',
    'cbv_mean', 'mtt_mean', 'tmax_mean', 'minimum_lvo_distance_mm',
    'lvo_overlap_fraction',
]
MODEL_VESSEL_FEATURE_NAMES = SOURCE_VESSEL_FEATURE_NAMES[1:]
SOURCE_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'atlas_mode_syn',
    'atlas_mode_affine', 'atlas_mode_disabled', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
MODEL_CASE_FEATURE_NAMES = [
    'atlas_enabled', 'registration_confidence', 'cow_prior_valid',
    'lvo_present', 'vessel_presence_fraction', 'perfusion_valid_fraction',
    'contralateral_pair_valid_fraction', 'log1p_brain_volume_ml',
]
FORBIDDEN_TARGET_KEYS = {
    'target_binary', 'target_soft', 'binary_loss_weight',
    'source_target_volume_ml', 'target_binary_volume_ml',
    'target_soft_volume_ml', 'target_crop_coverage',
}

TARGET_POLICY_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_graph_features_target_policy_v3.csv'
)
FOLD_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'trial8_train5fold_development_manifest_v3_subvoxel_aware.csv'
)
MULTIMODAL_MANIFEST_PATH = (
    PROJECT_ROOT / 'metadata'
    / 'model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv'
)
GRAPH_SCHEMA_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'vascular_graph_features'
    / 'v1_atlas30_vessel13_target_free' / 'verification'
    / 'v1_full_integrity_and_semantics'
    / 'graph_schema_verified_pickle_free.npz'
)
MULTIMODAL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_fold_clinical_scalers_multimodal_loader'
    / 'v2_multimodal_loader_gate.json'
)
V2_MODEL_GATE_PATH = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / 'trial8_vascufate_net_v2_asymmetric_vascular_fusion'
    / 'v1_zero_init_gated_fusion_construction'
    / 'v2_model_construction_gate.json'
)
BLOCK1_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / BLOCK1_VERSION
)
BLOCK1_GATE_PATH = BLOCK1_ROOT / 'v2_2_block1_preflight_gate.json'
BLOCK1_CONTRACT_PATH = BLOCK1_ROOT / 'v2_2_architecture_contract.json'
BLOCK1_COMPONENT_AUDIT_PATH = BLOCK1_ROOT / 'fold0_fit_component_audit.csv'

OUTPUT_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'architecture_preflight'
    / MODEL_VERSION / PREFLIGHT_VERSION
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ARCHITECTURE_PATH = OUTPUT_ROOT / 'v2_2_architecture_config.json'
PARAMETER_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_parameter_audit.csv'
OUTPUT_SHAPE_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_output_shape_audit.csv'
GRADIENT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_gradient_audit.csv'
ABLATION_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_controlled_ablation_audit.csv'
PREFLIGHT_AUDIT_PATH = OUTPUT_ROOT / 'v2_2_model_preflight_audit.csv'
PREFLIGHT_GATE_PATH = OUTPUT_ROOT / 'v2_2_model_construction_gate.json'






def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    return value


def atomic_write_json(record, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as handle:
        json.dump(to_builtin(record), handle, indent=2, sort_keys=True)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def decode_strings(array):
    return [
        value.decode('utf-8') if isinstance(value, bytes) else str(value)
        for value in np.asarray(array).reshape(-1).tolist()
    ]


def scalar_string(value):
    value = np.asarray(value).reshape(-1)[0]
    return value.decode('utf-8') if isinstance(value, bytes) else str(value)


def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return False
    normalized = str(value).strip().lower()
    if normalized in {'true', '1', 'yes', 'y'}:
        return True
    if normalized in {'false', '0', 'no', 'n', ''}:
        return False
    raise ValueError(f'Cannot parse boolean value: {value!r}')


def set_determinism(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parameter_count(module, trainable_only=False):
    parameters = module.parameters()
    if trainable_only:
        parameters = (p for p in parameters if p.requires_grad)
    return int(sum(p.numel() for p in parameters))


def gradient_norm(module):
    squared = 0.0
    observed = False
    finite = True
    for parameter in module.parameters():
        if parameter.grad is None:
            continue
        observed = True
        finite = finite and bool(torch.isfinite(parameter.grad).all())
        squared += float(parameter.grad.detach().float().pow(2).sum().cpu())
    return math.sqrt(squared), observed, finite


def symmetric_pad(array, target_shape, spatial_axes):
    output_shape = list(array.shape)
    starts = []
    for axis, target in zip(spatial_axes, target_shape):
        if array.shape[axis] > target:
            raise RuntimeError(
                f'Array shape {array.shape} exceeds target {target_shape}.'
            )
        output_shape[axis] = target
        starts.append((target - array.shape[axis]) // 2)
    output = np.zeros(output_shape, dtype=array.dtype)
    destination = [slice(None)] * array.ndim
    for axis, start, source_size in zip(
        spatial_axes, starts, [array.shape[a] for a in spatial_axes]
    ):
        destination[axis] = slice(start, start + source_size)
    output[tuple(destination)] = array
    return output, tuple(starts)


def crop_at(array, origin, output_shape, spatial_axes):
    slices = [slice(None)] * array.ndim
    for axis, start, size in zip(spatial_axes, origin, output_shape):
        slices[axis] = slice(int(start), int(start) + int(size))
    result = array[tuple(slices)]
    expected = list(array.shape)
    for axis, size in zip(spatial_axes, output_shape):
        expected[axis] = size
    if tuple(result.shape) != tuple(expected):
        raise RuntimeError(
            f'Crop shape {result.shape} does not match {tuple(expected)}.'
        )
    return result


def masked_scale(values, valid, mean, std, clip_value):
    transformed = (values - mean) / std
    transformed = np.clip(transformed, -clip_value, clip_value)
    return np.where(valid, transformed, 0.0).astype(np.float32)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def resolve_gate_mapping(mapping, fold):
    if str(fold) in mapping:
        return mapping[str(fold)]
    if fold in mapping:
        return mapping[fold]
    raise KeyError(f'Fold {fold} missing from gate mapping.')


def all_tensor_outputs_finite(output):
    return all(
        bool(torch.isfinite(value).all())
        for value in output.values()
        if isinstance(value, torch.Tensor) and value.dtype.is_floating_point
    )


def tensor_maximum_difference(first, second):
    return float((first.detach() - second.detach()).abs().max().cpu())






class BasisRelationBank(nn.Module):
    def __init__(self, num_relations, num_bases, heads, head_dim):
        super().__init__()
        self.bases = nn.Parameter(torch.empty(num_bases, heads, head_dim))
        self.coefficients = nn.Parameter(torch.empty(num_relations, num_bases))
        nn.init.xavier_uniform_(self.bases)
        nn.init.normal_(self.coefficients, mean=0.0, std=0.02)

    def forward(self):
        weights = torch.softmax(self.coefficients, dim=-1)
        return torch.einsum('rb,bhd->rhd', weights, self.bases)


class RelationalGATv2Layer(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        if hidden_dim % heads != 0:
            raise ValueError('hidden_dim must be divisible by heads.')
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.source_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.target_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(self, nodes, node_valid, edge_index, edge_type, edge_weight):
        batch_size, node_count, _ = nodes.shape
        source_index = edge_index[0]
        target_index = edge_index[1]
        source = self.source_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        target = self.target_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        value = self.value_projection(nodes).view(
            batch_size, node_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            source[:, source_index] + target[:, target_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, source_index] + relation[None]
        edge_valid = node_valid[:, source_index] & node_valid[:, target_index]
        aggregated = nodes.new_zeros(
            batch_size, node_count, self.heads, self.head_dim
        )
        for target_node in range(node_count):
            selected = target_index == target_node
            if not bool(selected.any()):
                continue
            selected_scores = scores[:, selected]
            selected_valid = edge_valid[:, selected]
            selected_scores = selected_scores.masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
        proposal = self.output_projection(aggregated.flatten(2))
        updated = self.update(
            proposal.reshape(-1, self.hidden_dim),
            nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, node_count, self.hidden_dim)
        output = self.norm(nodes + self.dropout(updated))
        return output * node_valid[..., None].to(output.dtype)


class CrossVesselTerritoryAttention(nn.Module):
    def __init__(
        self, hidden_dim, heads, num_relations, num_bases=3, dropout=0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads
        self.query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.value = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.relations = BasisRelationBank(
            num_relations, num_bases, heads, self.head_dim
        )
        self.attention_vector = nn.Parameter(torch.empty(heads, self.head_dim))
        self.output = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.attention_vector)

    def forward(
        self, vessel_nodes, territory_nodes, vessel_valid, territory_valid,
        edge_index, edge_type, edge_weight,
    ):
        batch_size, territory_count, _ = territory_nodes.shape
        vessel_count = vessel_nodes.shape[1]
        vessel_index = edge_index[0]
        territory_index = edge_index[1]
        query = self.query(territory_nodes).view(
            batch_size, territory_count, self.heads, self.head_dim
        )
        key = self.key(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        value = self.value(vessel_nodes).view(
            batch_size, vessel_count, self.heads, self.head_dim
        )
        relation = self.relations()[edge_type]
        score_state = F.leaky_relu(
            query[:, territory_index] + key[:, vessel_index] + relation[None],
            negative_slope=0.2,
        )
        scores = (score_state * self.attention_vector[None, None]).sum(-1)
        scores = scores + torch.log(edge_weight.clamp_min(1e-6))[None, :, None]
        messages = value[:, vessel_index] + relation[None]
        edge_valid = (
            vessel_valid[:, vessel_index]
            & territory_valid[:, territory_index]
        )
        aggregated = territory_nodes.new_zeros(
            batch_size, territory_count, self.heads, self.head_dim
        )
        has_message = torch.zeros(
            batch_size, territory_count, dtype=torch.bool,
            device=territory_nodes.device,
        )
        for target_node in range(territory_count):
            selected = territory_index == target_node
            if not bool(selected.any()):
                continue
            selected_valid = edge_valid[:, selected]
            selected_scores = scores[:, selected].masked_fill(
                ~selected_valid[..., None], -1e4
            )
            attention = torch.softmax(selected_scores, dim=1)
            attention = attention * selected_valid[..., None].to(attention.dtype)
            attention = attention / attention.sum(
                dim=1, keepdim=True
            ).clamp_min(1e-6)
            aggregated[:, target_node] = (
                attention[..., None] * messages[:, selected]
            ).sum(dim=1)
            has_message[:, target_node] = selected_valid.any(dim=1)
        proposal = self.output(aggregated.flatten(2))
        candidate = self.update(
            proposal.reshape(-1, self.hidden_dim),
            territory_nodes.reshape(-1, self.hidden_dim),
        ).view(batch_size, territory_count, self.hidden_dim)
        candidate = self.norm(territory_nodes + self.dropout(candidate))
        output = torch.where(has_message[..., None], candidate, territory_nodes)
        return output * territory_valid[..., None].to(output.dtype)


def masked_weighted_mean(nodes, weights):
    weights = weights.to(nodes.dtype)
    denominator = weights.sum(dim=1, keepdim=True).clamp_min(1e-6)
    return (nodes * weights[..., None]).sum(dim=1) / denominator


class HeterogeneousVascularGraphEncoder(nn.Module):
    def __init__(
        self, territory_input_dim, vessel_input_dim, case_input_dim, schema,
        hidden_dim=64, heads=2, layers=2, relation_bases=3, dropout=0.1,
    ):
        super().__init__()
        self.territory_input = nn.Sequential(
            nn.Linear(territory_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.vessel_input = nn.Sequential(
            nn.Linear(vessel_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        self.case_input = nn.Sequential(
            nn.Linear(case_input_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
        )
        territory_relations = len(
            decode_strings(schema['territory_relation_names'])
        )
        vessel_relations = len(decode_strings(schema['vessel_relation_names']))
        cross_relations = len(decode_strings(schema['cross_relation_names']))
        self.territory_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, territory_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.vessel_layers = nn.ModuleList([
            RelationalGATv2Layer(
                hidden_dim, heads, vessel_relations, relation_bases, dropout
            ) for _ in range(layers)
        ])
        self.cross_layer = CrossVesselTerritoryAttention(
            hidden_dim, heads, cross_relations, relation_bases, dropout
        )
        self.global_fusion = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.node_head = nn.Linear(hidden_dim, 1)
        for name, dtype in [
            ('territory_edge_index', torch.long),
            ('territory_edge_type', torch.long),
            ('territory_edge_weight', torch.float32),
            ('vessel_edge_index', torch.long),
            ('vessel_edge_type', torch.long),
            ('vessel_edge_weight', torch.float32),
            ('cross_edge_index', torch.long),
            ('cross_edge_type', torch.long),
            ('cross_edge_weight', torch.float32),
        ]:
            self.register_buffer(name, torch.as_tensor(schema[name], dtype=dtype))

    def forward(
        self, territory_input, vessel_input, case_input, territory_valid,
        vessel_valid, territory_reliability,
    ):
        territory = self.territory_input(territory_input)
        vessel = self.vessel_input(vessel_input)
        territory = territory * territory_valid[..., None].to(territory.dtype)
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        for layer in self.territory_layers:
            territory = layer(
                territory, territory_valid, self.territory_edge_index,
                self.territory_edge_type, self.territory_edge_weight,
            )
        for layer in self.vessel_layers:
            vessel = layer(
                vessel, vessel_valid, self.vessel_edge_index,
                self.vessel_edge_type, self.vessel_edge_weight,
            )
        territory = self.cross_layer(
            vessel, territory, vessel_valid, territory_valid,
            self.cross_edge_index, self.cross_edge_type,
            self.cross_edge_weight,
        )
        territory_pool = masked_weighted_mean(
            territory,
            territory_reliability * territory_valid.to(
                territory_reliability.dtype
            ),
        )
        vessel_pool = masked_weighted_mean(vessel, vessel_valid.to(vessel.dtype))
        case_context = self.case_input(case_input)
        global_context = self.global_fusion(
            torch.cat([territory_pool, vessel_pool, case_context], dim=-1)
        )
        node_logits = self.node_head(territory).squeeze(-1)
        node_logits = node_logits * territory_valid.to(node_logits.dtype)
        return territory, global_context, node_logits


def group_count(channels):
    for groups in [8, 4, 2, 1]:
        if channels % groups == 0:
            return groups
    return 1


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, 3, stride=stride, padding=1, bias=False
        )
        self.norm1 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, 3, padding=1, bias=False
        )
        self.norm2 = nn.GroupNorm(group_count(out_channels), out_channels)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout3d(dropout) if dropout > 0 else nn.Identity()
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Conv3d(
                in_channels, out_channels, 1, stride=stride, bias=False
            )
        )

    def forward(self, x):
        residual = self.skip(x)
        x = self.activation(self.norm1(self.conv1(x)))
        x = self.dropout(x)
        x = self.norm2(self.conv2(x))
        return self.activation(x + residual)


class DecoderStage3D(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_channels, out_channels, 2, stride=2)
        self.block = ResidualBlock3D(
            out_channels + skip_channels, out_channels, dropout=dropout
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x, size=skip.shape[2:], mode='trilinear', align_corners=False
            )
        return self.block(torch.cat([x, skip], dim=1))


class GlobalFiLM3D(nn.Module):
    def __init__(self, context_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Sequential(
            nn.Linear(context_dim, context_dim * 2), nn.GELU(),
            nn.Linear(context_dim * 2, channels * 2),
        )
        nn.init.zeros_(self.project[-1].weight)
        nn.init.zeros_(self.project[-1].bias)

    def forward(self, feature, context):
        gamma, beta = self.project(context).chunk(2, dim=-1)
        gamma = self.bound * torch.tanh(gamma)[..., None, None, None]
        beta = self.bound * torch.tanh(beta)[..., None, None, None]
        return feature * (1.0 + gamma) + beta

    def activate_for_gradient_test(self, fraction_of_bound=0.25):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            
            
            
            
            
            
            final = self.project[-1]
            final.weight.zero_()
            final.bias.zero_()
            magnitude = math.atanh(float(fraction_of_bound))
            rows = torch.arange(final.out_features, device=final.weight.device)
            columns = rows.remainder(final.in_features)
            final.weight[rows, columns] = magnitude


class RegionalFiLM3D(nn.Module):
    def __init__(self, node_dim, channels, bound=0.1):
        super().__init__()
        self.bound = float(bound)
        self.project = nn.Linear(node_dim, channels * 2)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, feature, node_context, region_map):
        labels = F.interpolate(
            region_map[:, None].to(feature.dtype), size=feature.shape[2:],
            mode='nearest',
        )[:, 0].long().clamp(0, node_context.shape[1])
        parameters = self.project(node_context)
        padding = parameters.new_zeros(parameters.shape[0], 1, parameters.shape[-1])
        parameters = torch.cat([padding, parameters], dim=1)
        maps = []
        for batch_index in range(feature.shape[0]):
            gathered = parameters[batch_index][labels[batch_index]]
            maps.append(gathered.permute(3, 0, 1, 2))
        gamma, beta = torch.stack(maps, dim=0).chunk(2, dim=1)
        gamma = self.bound * torch.tanh(gamma)
        beta = self.bound * torch.tanh(beta)
        return feature * (1.0 + gamma) + beta


class ConfidenceGate(nn.Module):
    def __init__(self, confidence_dim, output_dim, bound=0.1, hidden_dim=32):
        super().__init__()
        self.bound = float(bound)
        self.network = nn.Sequential(
            nn.Linear(confidence_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    def forward(self, confidence):
        return self.bound * torch.tanh(self.network(confidence))

    def activate_for_gradient_test(self, fraction_of_bound=0.5):
        if not 0.0 < fraction_of_bound < 1.0:
            raise ValueError('fraction_of_bound must be in (0, 1).')
        with torch.no_grad():
            self.network[-1].bias.fill_(math.atanh(float(fraction_of_bound)))


class AsymmetricVascularStream3D(nn.Module):
    def __init__(self, in_channels=5, channels=(8, 16, 32, 48)):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.half_stage = ResidualBlock3D(c1, c2, stride=2)
        self.quarter = ResidualBlock3D(c2, c3, stride=2)
        self.eighth = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.sixteenth = ResidualBlock3D(c4, c4, stride=2, dropout=0.05)

    def forward(self, vascular_image):
        v1 = self.stem(vascular_image)
        v2 = self.half_stage(v1)
        v3 = self.quarter(v2)
        v4 = self.eighth(v3)
        v5 = self.sixteenth(v4)
        return v3, v4, v5, v5.mean(dim=(2, 3, 4))


class SpatialConfidenceFusion3D(nn.Module):
    def __init__(
        self, vascular_channels, image_channels, confidence_dim=4, bound=0.1
    ):
        super().__init__()
        self.project = nn.Conv3d(
            vascular_channels, image_channels, 1, bias=False
        )
        self.norm = nn.GroupNorm(group_count(image_channels), image_channels)
        self.gate = ConfidenceGate(confidence_dim, image_channels, bound=bound)

    def forward(self, image_feature, vascular_feature, confidence):
        if vascular_feature.shape[2:] != image_feature.shape[2:]:
            vascular_feature = F.interpolate(
                vascular_feature, size=image_feature.shape[2:],
                mode='trilinear', align_corners=False,
            )
        residual = self.norm(self.project(vascular_feature))
        gate = self.gate(confidence)[..., None, None, None]
        return image_feature + gate * residual, gate.flatten(1)


class ClinicalTokenEncoder(nn.Module):
    def __init__(self, input_dim=44, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, clinical_input):
        return self.encoder(clinical_input)


class OcclusionSourceEncoder(nn.Module):
    def __init__(self, value_dim=7, output_dim=32, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(value_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, output_dim),
            nn.LayerNorm(output_dim), nn.GELU(),
        )

    def forward(self, values, valid):
        valid = valid.to(values.dtype)
        return self.encoder(torch.cat([values * valid, valid], dim=-1))


class ContextConfidenceFusion(nn.Module):
    def __init__(
        self, vascular_dim, occlusion_dim, clinical_dim, output_dim=64,
        confidence_dim=4, bound=0.1, dropout=0.10,
    ):
        super().__init__()
        self.source = nn.Sequential(
            nn.Linear(vascular_dim + occlusion_dim + clinical_dim, 128),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim),
        )
        self.gate = ConfidenceGate(confidence_dim, output_dim, bound=bound)

    def forward(
        self, graph_context, vascular_context, occlusion_context,
        clinical_context, confidence,
    ):
        source_context = self.source(torch.cat([
            vascular_context, occlusion_context, clinical_context
        ], dim=-1))
        gate = self.gate(confidence)
        return graph_context + gate * source_context, source_context, gate


class VascuFateNetV2(nn.Module):
    def __init__(
        self, schema, image_channels=15, territory_feature_dim=65,
        vessel_feature_dim=15, case_feature_dim=8,
        channels=(32, 64, 128, 224, 320), graph_hidden=64,
        graph_heads=2, graph_layers=2, relation_bases=3,
        graph_dropout=0.1, atlas_dropout=0.2, film_bound=0.1,
        vascular_channels=(8, 16, 32, 48), clinical_input_dim=44,
        clinical_embed_dim=32, occlusion_value_dim=7,
        occlusion_embed_dim=32, confidence_dim=4, fusion_bound=0.1,
        vascular_dropout=0.15, clinical_dropout=0.20,
        occlusion_dropout=0.20,
    ):
        super().__init__()
        self.atlas_dropout = float(atlas_dropout)
        self.vascular_dropout = float(vascular_dropout)
        self.clinical_dropout = float(clinical_dropout)
        self.occlusion_dropout = float(occlusion_dropout)
        self.graph_encoder = HeterogeneousVascularGraphEncoder(
            territory_feature_dim * 2 + 1, vessel_feature_dim * 2,
            case_feature_dim, schema, graph_hidden, graph_heads, graph_layers,
            relation_bases, graph_dropout,
        )
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(image_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.vascular_stream = AsymmetricVascularStream3D(
            VASCULAR_INPUT_CHANNELS, vascular_channels
        )
        _, _, vc3, vc4 = vascular_channels
        self.vascular_fusion_quarter = SpatialConfidenceFusion3D(
            vc3, c3, confidence_dim, fusion_bound
        )
        self.vascular_fusion_eighth = SpatialConfidenceFusion3D(
            vc4, c4, confidence_dim, fusion_bound
        )
        self.vascular_fusion_bottleneck = SpatialConfidenceFusion3D(
            vc4, c5, confidence_dim, fusion_bound
        )
        self.clinical_encoder = ClinicalTokenEncoder(
            clinical_input_dim, clinical_embed_dim
        )
        self.occlusion_encoder = OcclusionSourceEncoder(
            occlusion_value_dim, occlusion_embed_dim
        )
        self.context_fusion = ContextConfidenceFusion(
            vc4, occlusion_embed_dim, clinical_embed_dim, graph_hidden,
            confidence_dim, fusion_bound,
        )
        self.global_film = GlobalFiLM3D(graph_hidden, c5, film_bound)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.regional_films = nn.ModuleList([
            RegionalFiLM3D(graph_hidden, c4, film_bound),
            RegionalFiLM3D(graph_hidden, c3, film_bound),
            RegionalFiLM3D(graph_hidden, c2, film_bound),
        ])
        self.segmentation_head = nn.Conv3d(c1, 1, 1)
        self.auxiliary_half_head = nn.Conv3d(c2, 1, 1)
        self.auxiliary_quarter_head = nn.Conv3d(c3, 1, 1)

    def _effective_atlas(self, atlas_enabled, force_atlas_dropout=None):
        atlas_enabled = atlas_enabled.bool()
        if force_atlas_dropout is True:
            dropped = torch.ones_like(atlas_enabled)
        elif force_atlas_dropout is False:
            dropped = torch.zeros_like(atlas_enabled)
        elif self.training and self.atlas_dropout > 0:
            dropped = torch.rand_like(atlas_enabled.float()) < self.atlas_dropout
        else:
            dropped = torch.zeros_like(atlas_enabled)
        return atlas_enabled & ~dropped, dropped

    def _effective_branch(self, available, probability, forced):
        available = available.bool()
        if forced is True:
            dropped = torch.ones_like(available)
        elif forced is False:
            dropped = torch.zeros_like(available)
        elif self.training and probability > 0:
            dropped = torch.rand_like(available.float()) < probability
        else:
            dropped = torch.zeros_like(available)
        return available & ~dropped, dropped

    def activate_new_fusion_gates_for_gradient_test(self):
        for module in [
            self.vascular_fusion_quarter.gate,
            self.vascular_fusion_eighth.gate,
            self.vascular_fusion_bottleneck.gate,
            self.context_fusion.gate,
        ]:
            module.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, force_atlas_dropout=None,
        force_vascular_dropout=None, force_occlusion_dropout=None,
        force_clinical_dropout=None,
    ):
        effective_atlas, atlas_dropped = self._effective_atlas(
            atlas_enabled, force_atlas_dropout
        )
        effective_vascular, vascular_dropped = self._effective_branch(
            vascular_available, self.vascular_dropout, force_vascular_dropout
        )
        effective_occlusion, occlusion_dropped = self._effective_branch(
            occlusion_available, self.occlusion_dropout,
            force_occlusion_dropout,
        )
        effective_clinical, clinical_dropped = self._effective_branch(
            clinical_available, self.clinical_dropout, force_clinical_dropout
        )
        effective_territory_valid = (
            territory_valid.bool() & effective_atlas[:, None]
        )
        effective_reliability = territory_reliability * effective_atlas[
            :, None
        ].to(territory_reliability.dtype)
        effective_region_map = (
            region_map * effective_atlas[:, None, None, None].long()
        )
        effective_territory_features = territory_features * (
            effective_territory_valid[..., None].to(territory_features.dtype)
        )
        effective_feature_valid = territory_feature_valid * (
            effective_territory_valid[..., None].to(
                territory_feature_valid.dtype
            )
        )
        territory_input = torch.cat([
            effective_territory_features, effective_feature_valid,
            effective_reliability[..., None],
        ], dim=-1)
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        effective_case_features = case_features.clone()
        atlas_gate = effective_atlas.to(case_features.dtype)
        effective_case_features[..., 0] *= atlas_gate
        effective_case_features[..., 1] *= atlas_gate
        territory_context, global_context, node_logits = self.graph_encoder(
            territory_input, vessel_input, effective_case_features,
            effective_territory_valid, vessel_valid.bool(),
            effective_reliability,
        )
        vascular_gate = effective_vascular.to(image.dtype)
        v3, v4, v5, vascular_context = self.vascular_stream(
            vascular_image * vascular_gate[:, None, None, None, None]
        )
        vascular_context = vascular_context * vascular_gate[:, None]
        safe_occlusion_valid = occlusion_valid * effective_occlusion[
            :, None
        ].to(occlusion_valid.dtype)
        occlusion_context = self.occlusion_encoder(
            occlusion_values, safe_occlusion_valid
        ) * effective_occlusion[:, None].to(image.dtype)
        clinical_context = self.clinical_encoder(clinical_input) * (
            effective_clinical[:, None].to(image.dtype)
        )
        effective_confidence = confidence_features.clone()
        effective_confidence[:, 0] *= vascular_gate
        effective_confidence[:, 1] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 2] *= effective_atlas.to(image.dtype)
        effective_confidence[:, 3] *= effective_occlusion.to(image.dtype)
        fused_context, source_context, context_gate = self.context_fusion(
            global_context, vascular_context, occlusion_context,
            clinical_context, effective_confidence,
        )
        e1 = self.stem(image)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e3, quarter_gate = self.vascular_fusion_quarter(
            e3, v3, effective_confidence
        )
        e4 = self.encoder4(e3)
        e4, eighth_gate = self.vascular_fusion_eighth(
            e4, v4, effective_confidence
        )
        bottleneck = self.bottleneck(e4)
        bottleneck, bottleneck_gate = self.vascular_fusion_bottleneck(
            bottleneck, v5, effective_confidence
        )
        bottleneck = self.global_film(bottleneck, fused_context)
        d4 = self.decoder4(bottleneck, e4)
        d4 = self.regional_films[0](
            d4, territory_context, effective_region_map
        )
        d3 = self.decoder3(d4, e3)
        d3 = self.regional_films[1](
            d3, territory_context, effective_region_map
        )
        d2 = self.decoder2(d3, e2)
        d2 = self.regional_films[2](
            d2, territory_context, effective_region_map
        )
        d1 = self.decoder1(d2, e1)
        return {
            'logits': self.segmentation_head(d1),
            'auxiliary_half_logits': self.auxiliary_half_head(d2),
            'auxiliary_quarter_logits': self.auxiliary_quarter_head(d3),
            'territory_node_logits': node_logits,
            'global_graph_context': global_context,
            'fused_global_context': fused_context,
            'new_source_context': source_context,
            'context_fusion_gate': context_gate,
            'vascular_quarter_gate': quarter_gate,
            'vascular_eighth_gate': eighth_gate,
            'vascular_bottleneck_gate': bottleneck_gate,
            'effective_atlas_enabled': effective_atlas,
            'atlas_dropped': atlas_dropped,
            'effective_vascular_enabled': effective_vascular,
            'vascular_dropped': vascular_dropped,
            'effective_occlusion_enabled': effective_occlusion,
            'occlusion_dropped': occlusion_dropped,
            'effective_clinical_enabled': effective_clinical,
            'clinical_dropped': clinical_dropped,
        }






class DirectedArterialProjector(nn.Module):
    

    def __init__(self, schema, vessel_feature_dim=15, hidden_dim=64):
        super().__init__()
        relation_names = decode_strings(schema['cross_relation_names'])
        if relation_names != [
            'primary_supply', 'communicating_or_proximal_supply'
        ]:
            raise RuntimeError(
                'Directed arterial relation order changed: '
                f'{relation_names}'
            )
        self.vessel_encoder = nn.Sequential(
            nn.Linear(vessel_feature_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.relation_embedding = nn.Embedding(len(relation_names), hidden_dim)
        self.message = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.territory_update = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
        )
        self.risk_head = nn.Linear(hidden_dim, 1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        edge_type = np.asarray(schema['cross_edge_type'], dtype=np.int64)
        edge_weight = np.asarray(
            schema['cross_edge_weight'], dtype=np.float32
        )
        relation_prior = np.asarray([1.0, 0.5], dtype=np.float32)
        directed_prior = edge_weight * relation_prior[edge_type]
        self.register_buffer(
            'edge_index', torch.as_tensor(
                schema['cross_edge_index'], dtype=torch.long
            )
        )
        self.register_buffer(
            'edge_type', torch.as_tensor(edge_type, dtype=torch.long)
        )
        self.register_buffer(
            'edge_prior', torch.as_tensor(directed_prior, dtype=torch.float32)
        )

    def forward(
        self, vessel_features, vessel_feature_valid, vessel_valid,
        territory_valid, territory_reliability, atlas_enabled,
    ):
        vessel_input = torch.cat(
            [vessel_features, vessel_feature_valid], dim=-1
        )
        vessel = self.vessel_encoder(vessel_input)
        vessel_valid = vessel_valid.bool()
        territory_valid = (
            territory_valid.bool() & atlas_enabled.bool()[:, None]
        )
        vessel = vessel * vessel_valid[..., None].to(vessel.dtype)
        source = self.edge_index[0]
        target = self.edge_index[1]
        relation = self.relation_embedding(self.edge_type)
        messages = self.message(torch.cat([
            vessel[:, source], relation[None].expand(vessel.shape[0], -1, -1)
        ], dim=-1))
        edge_valid = vessel_valid[:, source] & territory_valid[:, target]
        edge_weight = self.edge_prior[None] * edge_valid.to(messages.dtype)
        aggregated = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, messages.shape[-1]
        )
        denominator = messages.new_zeros(
            messages.shape[0], NUM_TERRITORY_NODES, 1
        )
        for territory_index in range(NUM_TERRITORY_NODES):
            selected = target == territory_index
            if not bool(selected.any()):
                continue
            weights = edge_weight[:, selected]
            aggregated[:, territory_index] = (
                messages[:, selected] * weights[..., None]
            ).sum(dim=1)
            denominator[:, territory_index, 0] = weights.sum(dim=1)
        has_message = denominator[..., 0] > 0
        aggregated = aggregated / denominator.clamp_min(1e-6)
        territory_context = self.territory_update(aggregated)
        territory_context = territory_context * has_message[..., None].to(
            territory_context.dtype
        )
        effective_weight = (
            territory_reliability * has_message.to(territory_reliability.dtype)
        )
        global_context = self.global_head(
            masked_weighted_mean(territory_context, effective_weight)
        )
        risk_logits = self.risk_head(territory_context).squeeze(-1)
        risk_logits = risk_logits * has_message.to(risk_logits.dtype)
        return {
            'territory_context': territory_context,
            'territory_risk_logits': risk_logits,
            'territory_projection_valid': has_message,
            'arterial_global_context': global_context,
        }


class GlobalProposalStream3D(nn.Module):
    

    def __init__(self, in_channels=17, channels=GLOBAL_CHANNELS):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.stem = ResidualBlock3D(in_channels, c1)
        self.encoder2 = ResidualBlock3D(c1, c2, stride=2)
        self.encoder3 = ResidualBlock3D(c2, c3, stride=2)
        self.encoder4 = ResidualBlock3D(c3, c4, stride=2, dropout=0.05)
        self.bottleneck = ResidualBlock3D(c4, c5, stride=2, dropout=0.10)
        self.context_film = GlobalFiLM3D(GLOBAL_CONTEXT_DIM, c5, bound=0.10)
        self.decoder4 = DecoderStage3D(c5, c4, c4, dropout=0.05)
        self.decoder3 = DecoderStage3D(c4, c3, c3, dropout=0.05)
        self.decoder2 = DecoderStage3D(c3, c2, c2)
        self.decoder1 = DecoderStage3D(c2, c1, c1)
        self.proposal_head = nn.Conv3d(c1, 1, 1)

    def encode(self, x):
        e1 = self.stem(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)
        bottleneck = self.bottleneck(e4)
        pooled = bottleneck.mean(dim=(2, 3, 4))
        return (e1, e2, e3, e4), bottleneck, pooled

    def decode(self, skips, bottleneck, context):
        e1, e2, e3, e4 = skips
        bottleneck = self.context_film(bottleneck, context)
        d4 = self.decoder4(bottleneck, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        return self.proposal_head(d1), d1


class GlobalBurdenContextRouter(nn.Module):
    

    def __init__(
        self, global_image_dim=96, arterial_dim=64, clinical_dim=44,
        occlusion_dim=7, confidence_dim=4, output_dim=64,
    ):
        super().__init__()
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.20), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.occlusion_encoder = nn.Sequential(
            nn.Linear(occlusion_dim * 2, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Dropout(0.15), nn.Linear(64, 32), nn.LayerNorm(32), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(
                global_image_dim + arterial_dim + 32 + 32
                + confidence_dim + 3,
                128,
            ),
            nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(128, output_dim), nn.LayerNorm(output_dim), nn.GELU(),
        )
        self.presence_head = nn.Linear(output_dim, 1)
        self.log_volume_head = nn.Linear(output_dim, 1)

    def forward(
        self, global_image_context, arterial_context, clinical_input,
        occlusion_values, occlusion_valid, confidence_features,
        atlas_available, occlusion_available, clinical_available,
    ):
        clinical_gate = clinical_available.to(global_image_context.dtype)[:, None]
        occlusion_gate = occlusion_available.to(
            global_image_context.dtype
        )[:, None]
        clinical_context = self.clinical_encoder(clinical_input) * clinical_gate
        occlusion_valid = occlusion_valid * occlusion_gate
        occlusion_context = self.occlusion_encoder(torch.cat([
            occlusion_values * occlusion_valid, occlusion_valid
        ], dim=-1)) * occlusion_gate
        availability = torch.stack([
            atlas_available.to(global_image_context.dtype),
            occlusion_available.to(global_image_context.dtype),
            clinical_available.to(global_image_context.dtype),
        ], dim=-1)
        context = self.fusion(torch.cat([
            global_image_context, arterial_context, clinical_context,
            occlusion_context, confidence_features, availability,
        ], dim=-1))
        return {
            'global_context': context,
            'clinical_context': clinical_context,
            'occlusion_context': occlusion_context,
            'lesion_presence_logit': self.presence_head(context).squeeze(-1),
            'log_volume_prediction': self.log_volume_head(context).squeeze(-1),
        }


def territory_values_to_spatial(values, valid, region_map):
    labels = region_map.long().clamp(0, values.shape[1])
    padded_values = torch.cat([
        values.new_zeros(values.shape[0], 1), values
    ], dim=1)
    padded_valid = torch.cat([
        valid.new_zeros(valid.shape[0], 1), valid
    ], dim=1)
    value_maps = []
    valid_maps = []
    for batch_index in range(values.shape[0]):
        value_maps.append(padded_values[batch_index][labels[batch_index]])
        valid_maps.append(padded_valid[batch_index][labels[batch_index]])
    return torch.stack(value_maps, dim=0), torch.stack(valid_maps, dim=0)


def local_sampling_grid(local_origin, canvas_shape, local_shape, device, dtype):
    axes = []
    for axis in range(3):
        coordinate = (
            local_origin[:, axis:axis + 1].to(dtype)
            + torch.arange(local_shape[axis], device=device, dtype=dtype)[None]
            + 0.5
        )
        axes.append(2.0 * coordinate / float(canvas_shape[axis]) - 1.0)
    gx = axes[0][:, :, None, None].expand(-1, -1, local_shape[1], local_shape[2])
    gy = axes[1][:, None, :, None].expand(-1, local_shape[0], -1, local_shape[2])
    gz = axes[2][:, None, None, :].expand(-1, local_shape[0], local_shape[1], -1)
    return torch.stack([gz, gy, gx], dim=-1)


def sample_global_to_local(global_tensor, local_origin, canvas_shape, local_shape):
    grid = local_sampling_grid(
        local_origin, canvas_shape, local_shape, global_tensor.device,
        global_tensor.dtype,
    )
    return F.grid_sample(
        global_tensor, grid, mode='bilinear', padding_mode='border',
        align_corners=False,
    )


class ProposalConditionedInputFusion(nn.Module):
    

    def __init__(
        self, global_feature_channels=12, image_channels=15,
        confidence_dim=5, bound=0.10,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(global_feature_channels + 2, 24, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            nn.Conv3d(24, image_channels, 1, bias=False),
        )
        self.gate = ConfidenceGate(
            confidence_dim, image_channels, bound=bound, hidden_dim=32
        )

    def forward(
        self, image, proposal_local, feature_local, arterial_local,
        confidence, enabled,
    ):
        residual = self.project(torch.cat([
            torch.sigmoid(proposal_local), feature_local, arterial_local
        ], dim=1))
        gate = self.gate(confidence) * enabled[:, None].to(image.dtype)
        return image + gate[..., None, None, None] * residual, gate


class VascuFateNetV22(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.local_model = VascuFateNetV2(schema)
        self.arterial_projector = DirectedArterialProjector(schema)
        self.global_stream = GlobalProposalStream3D()
        self.global_router = GlobalBurdenContextRouter()
        self.local_fusion = ProposalConditionedInputFusion()
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def activate_new_gates_for_gradient_test(self):
        self.local_model.activate_new_fusion_gates_for_gradient_test()
        self.global_stream.context_film.activate_for_gradient_test(0.25)
        self.local_fusion.gate.activate_for_gradient_test(0.5)

    def forward(
        self, image, region_map, territory_features, territory_feature_valid,
        territory_valid, territory_reliability, vessel_features,
        vessel_feature_valid, vessel_valid, case_features, atlas_enabled,
        vascular_image, confidence_features, occlusion_values,
        occlusion_valid, clinical_input, vascular_available,
        occlusion_available, clinical_available, global_image,
        global_region_map, local_origin_in_canvas,
        force_atlas_dropout=False, force_vascular_dropout=False,
        force_occlusion_dropout=False, force_clinical_dropout=False,
        force_global_dropout=False, force_arterial_dropout=False,
    ):
        effective_atlas = atlas_enabled.bool() & ~torch.full_like(
            atlas_enabled.bool(), bool(force_atlas_dropout)
        )
        effective_arterial = effective_atlas & ~torch.full_like(
            effective_atlas, bool(force_arterial_dropout)
        )
        effective_clinical = clinical_available.bool() & ~torch.full_like(
            clinical_available.bool(), bool(force_clinical_dropout)
        )
        effective_occlusion = occlusion_available.bool() & ~torch.full_like(
            occlusion_available.bool(), bool(force_occlusion_dropout)
        )
        safe_vessel_valid = vessel_valid.bool() & effective_arterial[:, None]
        arterial = self.arterial_projector(
            vessel_features, vessel_feature_valid, safe_vessel_valid,
            territory_valid, territory_reliability, effective_arterial,
        )
        arterial_map, arterial_map_valid = territory_values_to_spatial(
            arterial['territory_risk_logits'],
            arterial['territory_projection_valid'], global_region_map,
        )
        arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
        global_input = torch.cat([
            global_image, arterial_map[:, None],
            arterial_map_valid[:, None].to(global_image.dtype),
        ], dim=1)
        skips, bottleneck, global_image_context = self.global_stream.encode(
            global_input
        )
        routed = self.global_router(
            global_image_context,
            arterial['arterial_global_context']
            * effective_arterial[:, None].to(global_image.dtype),
            clinical_input * effective_clinical[:, None].to(global_image.dtype),
            occlusion_values, occlusion_valid, confidence_features,
            effective_atlas, effective_occlusion, effective_clinical,
        )
        proposal_logits, global_feature = self.global_stream.decode(
            skips, bottleneck, routed['global_context']
        )
        proposal_local = sample_global_to_local(
            proposal_logits, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        feature_local = sample_global_to_local(
            global_feature, local_origin_in_canvas, GLOBAL_CANVAS_SHAPE,
            LOCAL_PATCH_SIZE,
        )
        arterial_local = sample_global_to_local(
            arterial_map[:, None], local_origin_in_canvas,
            GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
        )
        global_enabled = ~torch.full_like(
            atlas_enabled.bool(), bool(force_global_dropout)
        )
        local_confidence = torch.cat([
            confidence_features,
            torch.sigmoid(routed['lesion_presence_logit'])[:, None],
        ], dim=-1)
        conditioned_image, proposal_gate = self.local_fusion(
            image, proposal_local, feature_local, arterial_local,
            local_confidence, global_enabled,
        )
        local = self.local_model(
            image=conditioned_image,
            region_map=region_map,
            territory_features=territory_features,
            territory_feature_valid=territory_feature_valid,
            territory_valid=territory_valid,
            territory_reliability=territory_reliability,
            vessel_features=vessel_features,
            vessel_feature_valid=vessel_feature_valid,
            vessel_valid=vessel_valid,
            case_features=case_features,
            atlas_enabled=atlas_enabled,
            vascular_image=vascular_image,
            confidence_features=confidence_features,
            occlusion_values=occlusion_values,
            occlusion_valid=occlusion_valid,
            clinical_input=clinical_input,
            vascular_available=vascular_available,
            occlusion_available=occlusion_available,
            clinical_available=clinical_available,
            force_atlas_dropout=force_atlas_dropout,
            force_vascular_dropout=force_vascular_dropout,
            force_occlusion_dropout=force_occlusion_dropout,
            force_clinical_dropout=force_clinical_dropout,
        )
        return {
            **local,
            'global_proposal_logits': proposal_logits,
            'global_proposal_local': proposal_local,
            'global_feature_local': feature_local,
            'arterial_territory_logits': arterial['territory_risk_logits'],
            'arterial_territory_valid': arterial[
                'territory_projection_valid'
            ],
            'arterial_global_context': arterial['arterial_global_context'],
            'arterial_global_map': arterial_map,
            'arterial_global_map_valid': arterial_map_valid,
            'lesion_presence_logit': routed['lesion_presence_logit'],
            'log_volume_prediction': routed['log_volume_prediction'],
            'global_burden_context': routed['global_context'],
            'proposal_fusion_gate': proposal_gate,
            'effective_global_enabled': global_enabled,
            'effective_arterial_enabled': effective_arterial,
        }






def load_graph_scaler(multimodal_gate, fold):
    path = Path(resolve_gate_mapping(multimodal_gate['graph_scaler_paths'], fold))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['graph_scaler_sha256'], fold
    )
    if file_sha256(path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} graph scaler hash mismatch.')
    with np.load(path, allow_pickle=False) as data:
        return {name: np.asarray(data[name]) for name in data.files}


def load_clinical_token(row, multimodal_gate, fold):
    sidecar_path = Path(str(row['trial8_clinical_sidecar_path']))
    if file_sha256(sidecar_path) != str(
        row['trial8_clinical_sidecar_sha256']
    ):
        raise RuntimeError(f"{row['subject_id']}: clinical sidecar changed.")
    with np.load(sidecar_path, allow_pickle=False) as data:
        if any(data[name].dtype.kind == 'O' for name in data.files):
            raise RuntimeError('Clinical sidecar contains object arrays.')
        values = np.asarray(data['values'], dtype=np.float32)
        valid = np.asarray(data['valid']) > 0
        feature_names = decode_strings(data['feature_names'])
    scaler_path = Path(resolve_gate_mapping(
        multimodal_gate['clinical_scaler_paths'], fold
    ))
    expected_hash = resolve_gate_mapping(
        multimodal_gate['clinical_scaler_sha256'], fold
    )
    if file_sha256(scaler_path) != expected_hash:
        raise RuntimeError(f'Fold-{fold} clinical scaler hash mismatch.')
    with np.load(scaler_path, allow_pickle=False) as data:
        scaler = {name: np.asarray(data[name]) for name in data.files}
    if feature_names != decode_strings(scaler['feature_names']):
        raise RuntimeError('Clinical sidecar/scaler feature order mismatch.')
    imputed = values.copy()
    imputed[~valid] = scaler['imputation'][~valid]
    transformed = imputed.astype(np.float32)
    scale_mask = np.asarray(scaler['scale_mask']) > 0
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    transformed[scale_mask] = np.clip(
        (transformed[scale_mask] - scaler['mean'][scale_mask])
        / scaler['std'][scale_mask],
        -clip_value, clip_value,
    )
    transformed[~scale_mask & ~valid] = 0.0
    token = np.concatenate([
        transformed, (~valid).astype(np.float32)
    ]).astype(np.float32)
    if token.shape != (CLINICAL_INPUT_DIM,) or not np.isfinite(token).all():
        raise RuntimeError('Clinical token is invalid.')
    return token, valid


def load_v22_subject(row, schema, multimodal_gate, local_origin=None):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    sidecar_path = Path(str(row['trial8_graph_sidecar_path']))
    if file_sha256(npz_path) != str(row['trial8_npz_sha256']):
        raise RuntimeError(f'{subject_id}: source NPZ hash mismatch.')
    if file_sha256(sidecar_path) != str(
        row['trial8_graph_sidecar_sha256']
    ):
        raise RuntimeError(f'{subject_id}: graph sidecar hash mismatch.')
    with np.load(npz_path, allow_pickle=False) as data:
        required = {
            'image', 'input_channels', 'cow_soft_union', 'lvo_proximity',
            'lvo_present',
        }
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(
                f'{subject_id}: source NPZ missing {missing}.'
            )
        image = np.asarray(data['image'], dtype=np.float32)
        input_channels = decode_strings(data['input_channels'])
        cow_soft_union = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo_proximity = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if image.ndim != 4 or image.shape[0] != 15:
        raise RuntimeError(f'{subject_id}: expected image [15, X, Y, Z].')
    if input_channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: input channel schema changed.')
    if not np.isfinite(image).all():
        raise RuntimeError(f'{subject_id}: image is non-finite.')
    if cow_soft_union.shape != image.shape[1:] or lvo_proximity.shape != image.shape[1:]:
        raise RuntimeError(f'{subject_id}: spatial-prior shape mismatch.')

    with np.load(sidecar_path, allow_pickle=False) as data:
        if FORBIDDEN_TARGET_KEYS & set(data.files):
            raise RuntimeError(f'{subject_id}: graph sidecar contains targets.')
        node_features = np.asarray(data['node_features'], dtype=np.float32)
        node_feature_valid = np.asarray(data['node_feature_valid']) > 0
        node_valid = np.asarray(data['node_valid']) > 0
        node_reliability = np.asarray(
            data['node_reliability'], dtype=np.float32
        )
        vessel_source = np.asarray(data['vessel_features'], dtype=np.float32)
        vessel_source_valid = np.asarray(data['vessel_feature_valid']) > 0
        vessel_valid = np.asarray(data['vessel_presence']) > 0
        case_source = np.asarray(data['case_features'], dtype=np.float32)
    expected_shapes = {
        'node_features': node_features.shape == (30, 65),
        'node_feature_valid': node_feature_valid.shape == (30, 65),
        'node_valid': node_valid.shape == (30,),
        'node_reliability': node_reliability.shape == (30,),
        'vessel_source': vessel_source.shape == (13, 16),
        'vessel_source_valid': vessel_source_valid.shape == (13, 16),
        'vessel_valid': vessel_valid.shape == (13,),
        'case_source': case_source.shape == (11,),
    }
    if not all(expected_shapes.values()):
        failed = [name for name, passed in expected_shapes.items() if not passed]
        raise RuntimeError(
            f'{subject_id}: graph sidecar shape mismatch: {failed}'
        )
    scaler = load_graph_scaler(multimodal_gate, RUNTIME_FOLD)
    vessel_indices = np.asarray(scaler['vessel_source_indices'], dtype=np.int64)
    case_indices = np.asarray(scaler['case_source_indices'], dtype=np.int64)
    clip_value = float(np.asarray(scaler['clip_value']).reshape(-1)[0])
    expected_vessel_indices = np.asarray([
        SOURCE_VESSEL_FEATURE_NAMES.index(name)
        for name in MODEL_VESSEL_FEATURE_NAMES
    ])
    expected_case_indices = np.asarray([
        SOURCE_CASE_FEATURE_NAMES.index(name)
        for name in MODEL_CASE_FEATURE_NAMES
    ])
    if not np.array_equal(vessel_indices, expected_vessel_indices):
        raise RuntimeError('Vessel scaler source indices changed.')
    if not np.array_equal(case_indices, expected_case_indices):
        raise RuntimeError('Case scaler source indices changed.')
    scaled_territory = masked_scale(
        node_features, node_feature_valid & node_valid[:, None],
        scaler['territory_mean'][None], scaler['territory_std'][None],
        clip_value,
    )
    selected_vessel = vessel_source[:, vessel_indices]
    selected_vessel_valid = (
        vessel_source_valid[:, vessel_indices] & vessel_valid[:, None]
    )
    scaled_vessel = masked_scale(
        selected_vessel, selected_vessel_valid,
        scaler['vessel_mean'][None], scaler['vessel_std'][None],
        clip_value,
    )
    selected_case = case_source[case_indices].astype(np.float32)
    case_scale_mask = np.asarray(scaler['case_scale_mask']) > 0
    scaled_case = selected_case.copy()
    scaled_case[case_scale_mask] = np.clip(
        (scaled_case[case_scale_mask] - scaler['case_mean'][case_scale_mask])
        / scaler['case_std'][case_scale_mask],
        -clip_value, clip_value,
    )

    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        atlas_path = Path(str(row['trial8_atlas_vascular30_path']))
        region_map = np.asanyarray(nib.load(str(atlas_path)).dataobj)
        if region_map.ndim == 4 and region_map.shape[-1] == 1:
            region_map = region_map[..., 0]
        region_map = np.rint(region_map).astype(np.int64)
        if region_map.shape != image.shape[1:]:
            raise RuntimeError(f'{subject_id}: atlas/image shape mismatch.')
        if region_map.min() < 0 or region_map.max() > 30:
            raise RuntimeError(f'{subject_id}: atlas labels outside 0..30.')
    else:
        region_map = np.zeros(image.shape[1:], dtype=np.int64)

    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region_map, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow_soft_union, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo_proximity, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: symmetric padding starts disagree.')
    if local_origin is None:
        local_origin = tuple(
            (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
            for i in range(3)
        )
    local_origin = tuple(int(v) for v in local_origin)
    for axis in range(3):
        if not 0 <= local_origin[axis] <= (
            GLOBAL_CANVAS_SHAPE[axis] - LOCAL_PATCH_SIZE[axis]
        ):
            raise RuntimeError(f'{subject_id}: invalid local crop origin.')
    image_patch = crop_at(
        image_canvas, local_origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    )
    region_patch = crop_at(
        region_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    cow_patch = crop_at(
        cow_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    lvo_patch = crop_at(
        lvo_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_image = F.avg_pool3d(
        torch.from_numpy(image_canvas[None]), kernel_size=GLOBAL_DOWNSAMPLE,
        stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    global_region_map = F.interpolate(
        torch.from_numpy(region_canvas[None, None].astype(np.float32)),
        size=GLOBAL_INPUT_SHAPE, mode='nearest',
    )[0, 0].numpy().astype(np.int64)
    vascular_patch = np.stack([
        image_patch[input_channels.index('cta')],
        image_patch[input_channels.index('tmax')],
        image_patch[input_channels.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    cow_prior_valid = bool(
        case_source[SOURCE_CASE_FEATURE_NAMES.index('cow_prior_valid')] > 0.5
    )
    registration_confidence = float(
        case_source[
            SOURCE_CASE_FEATURE_NAMES.index('registration_confidence')
        ]
    )
    confidence_features = np.asarray([
        float(vascular_patch[2].mean()), float(cow_prior_valid),
        float(np.clip(registration_confidence, 0.0, 1.0)),
        float(lvo_present),
    ], dtype=np.float32)
    occlusion_feature_names = [
        'cta_mean', 'cta_p90', 'tmax_mean', 'minimum_lvo_distance_mm',
        'lvo_overlap_fraction',
    ]
    occlusion_values = [float(lvo_present), float(lvo_patch.max())]
    occlusion_valid_values = [1.0, float(lvo_present)]
    for feature_name in occlusion_feature_names:
        feature_index = MODEL_VESSEL_FEATURE_NAMES.index(feature_name)
        valid_nodes = selected_vessel_valid[:, feature_index]
        if bool(valid_nodes.any()):
            occlusion_values.append(float(
                scaled_vessel[valid_nodes, feature_index].mean()
            ))
            occlusion_valid_values.append(1.0)
        else:
            occlusion_values.append(0.0)
            occlusion_valid_values.append(0.0)
    clinical_token, clinical_valid = load_clinical_token(
        row, multimodal_gate, RUNTIME_FOLD
    )
    output = {
        'image': image_patch.astype(np.float32),
        'region_map': region_patch.astype(np.int64),
        'territory_features': scaled_territory.astype(np.float32),
        'territory_feature_valid': node_feature_valid.astype(np.float32),
        'territory_valid': node_valid.astype(bool),
        'territory_reliability': node_reliability.astype(np.float32),
        'vessel_features': scaled_vessel.astype(np.float32),
        'vessel_feature_valid': selected_vessel_valid.astype(np.float32),
        'vessel_valid': vessel_valid.astype(bool),
        'case_features': scaled_case.astype(np.float32),
        'atlas_enabled': np.asarray(atlas_enabled, dtype=bool),
        'vascular_image': vascular_patch,
        'confidence_features': confidence_features,
        'occlusion_values': np.asarray(occlusion_values, dtype=np.float32),
        'occlusion_valid': np.asarray(
            occlusion_valid_values, dtype=np.float32
        ),
        'clinical_input': clinical_token,
        'vascular_available': np.asarray(True, dtype=bool),
        'occlusion_available': np.asarray(lvo_present, dtype=bool),
        'clinical_available': np.asarray(clinical_valid.any(), dtype=bool),
        'global_image': global_image,
        'global_region_map': global_region_map,
        'local_origin_in_canvas': np.asarray(local_origin, dtype=np.int64),
    }
    for name, value in output.items():
        value = np.asarray(value)
        if value.dtype.kind == 'f' and not np.isfinite(value).all():
            raise RuntimeError(f'{subject_id}: {name} is non-finite.')
    metadata = {
        'subject_id': subject_id,
        'npz_path': str(npz_path),
        'native_shape': list(image.shape[1:]),
        'pad_start': list(pad_start),
        'local_origin': list(local_origin),
        'atlas_enabled': atlas_enabled,
    }
    return output, metadata


def stack_sample(sample):
    return {
        key: torch.from_numpy(np.asarray(value)[None])
        for key, value in sample.items()
    }


def move_batch(batch, device):
    return {key: value.to(device) for key, value in batch.items()}


def load_smoke_target(row, local_origin):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        if 'target_binary' not in data.files or 'target_soft' not in data.files:
            raise RuntimeError(f'{subject_id}: target arrays unavailable.')
        target_binary = np.asarray(data['target_binary'], dtype=np.float32)
        target_soft = np.asarray(data['target_soft'], dtype=np.float32)
    target = np.maximum(target_binary, target_soft).astype(np.float32)
    target_canvas, _ = symmetric_pad(
        target, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    local_target = crop_at(
        target_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    global_target = F.max_pool3d(
        torch.from_numpy(target_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0, 0].numpy().astype(np.float32)
    if float(local_target.sum()) <= 0 or float(global_target.sum()) <= 0:
        raise RuntimeError(
            f'{subject_id}: smoke target not covered by local/global views.'
        )
    return {
        'local_target': local_target,
        'global_target': global_target,
        'target_voxels': int((target > 0).sum()),
        'local_support_voxels': int((local_target > 0).sum()),
        'global_support_voxels': int((global_target > 0).sum()),
        'target_arrays_read': True,
    }


def choose_smoke_subject(manifest_df):
    component_df = pd.read_csv(BLOCK1_COMPONENT_AUDIT_PATH)
    fit_components = component_df[
        component_df['subject_id'].astype(str).isin(
            manifest_df.loc[
                (manifest_df['dev_split'] == 'train')
                & (manifest_df['development_fold'] != FOLD0),
                'subject_id',
            ].astype(str)
        )
    ].copy()
    if fit_components.empty:
        raise RuntimeError('No fold-0 fit components available for smoke test.')
    chosen_component = fit_components.sort_values(
        ['component_volume_ml', 'subject_id'], ascending=[False, True]
    ).iloc[0]
    subject_id = str(chosen_component['subject_id'])
    row = manifest_df.set_index('subject_id').loc[subject_id]
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        target = np.maximum(
            np.asarray(data['target_binary'], dtype=np.float32),
            np.asarray(data['target_soft'], dtype=np.float32),
        )
        image_shape = np.asarray(data['image']).shape[1:]
    coordinates = np.argwhere(target > 0)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: selected smoke target is empty.')
    center_native = np.rint(np.median(coordinates, axis=0)).astype(int)
    pad_start = np.asarray([
        (GLOBAL_CANVAS_SHAPE[i] - image_shape[i]) // 2 for i in range(3)
    ])
    center_canvas = center_native + pad_start
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    origin = np.maximum(origin, 0)
    origin = np.minimum(
        origin, np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    )
    return subject_id, tuple(origin.astype(int).tolist())






def soft_dice_loss(logits, target, epsilon=1e-5):
    probability = torch.sigmoid(logits)
    intersection = (probability * target).sum()
    denominator = probability.sum() + target.sum()
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def territory_target_from_global(global_target, global_region_map):
    batch_size = global_target.shape[0]
    target = global_target.new_zeros(batch_size, NUM_TERRITORY_NODES)
    valid = torch.zeros_like(target, dtype=torch.bool)
    for batch_index in range(batch_size):
        labels = global_region_map[batch_index]
        for territory in range(1, NUM_TERRITORY_NODES + 1):
            mask = labels == territory
            if bool(mask.any()):
                valid[batch_index, territory - 1] = True
                target[batch_index, territory - 1] = global_target[
                    batch_index, 0
                ][mask].amax()
    return target, valid






def main():
    required_paths = [
        TARGET_POLICY_MANIFEST_PATH, FOLD_MANIFEST_PATH,
        MULTIMODAL_MANIFEST_PATH, GRAPH_SCHEMA_PATH, MULTIMODAL_GATE_PATH,
        V2_MODEL_GATE_PATH, BLOCK1_GATE_PATH, BLOCK1_CONTRACT_PATH,
        BLOCK1_COMPONENT_AUDIT_PATH,
    ]
    missing_paths = [str(path) for path in required_paths if not path.exists()]
    if missing_paths:
        raise FileNotFoundError(
            'Trial8 V2.2 Block 2 prerequisites are missing:\n'
            + '\n'.join(missing_paths)
        )
    input_hashes = {str(path): file_sha256(path) for path in required_paths}
    if input_hashes[str(TARGET_POLICY_MANIFEST_PATH)] != (
        EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ):
        raise RuntimeError('Target-policy manifest hash changed.')
    if input_hashes[str(FOLD_MANIFEST_PATH)] != EXPECTED_FOLD_MANIFEST_SHA256:
        raise RuntimeError('Five-fold manifest hash changed.')
    if input_hashes[str(MULTIMODAL_MANIFEST_PATH)] != (
        EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ):
        raise RuntimeError('Multimodal manifest hash changed.')
    if input_hashes[str(GRAPH_SCHEMA_PATH)] != EXPECTED_GRAPH_SCHEMA_SHA256:
        raise RuntimeError('Verified graph schema hash changed.')

    block1_gate = load_json(BLOCK1_GATE_PATH)
    block1_contract = load_json(BLOCK1_CONTRACT_PATH)
    multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
    v2_gate = load_json(V2_MODEL_GATE_PATH)
    upstream_checks = {
        'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
        'block1_architecture_implementation_authorized': (
            block1_gate.get('architecture_implementation_authorized') is True
        ),
        'block1_pilot_training_locked': (
            block1_gate.get('pilot_training_authorized') is False
        ),
        'block1_global_canvas_matches': tuple(
            block1_gate.get('global_canvas_shape', [])
        ) == GLOBAL_CANVAS_SHAPE,
        'block1_global_input_matches': tuple(
            block1_gate.get('global_input_shape', [])
        ) == GLOBAL_INPUT_SHAPE,
        'block1_directed_edges_are_68': int(
            block1_gate.get('directed_cross_edges', -1)
        ) == 68,
        'block1_fold0_fit_targets_only': (
            block1_gate.get('target_arrays_read_splits')
            == ['train_fold0_fit']
        ),
        'block1_heldout_validation_test_targets_sealed': (
            block1_gate.get('fold0_heldout_targets_read') is False
            and block1_gate.get('locked_validation_targets_read') is False
            and block1_gate.get('test_targets_read') is False
        ),
        'block1_parameter_budget_is_2p5m': int(
            block1_contract.get('parameter_budget', {}).get(
                'maximum_new_parameters', -1
            )
        ) == MAXIMUM_NEW_PARAMETERS,
        'multimodal_preflight_passed': (
            multimodal_gate.get('preflight_passed') is True
        ),
        'multimodal_training_locked': (
            multimodal_gate.get('fold0_pilot_training_authorized') is False
            and multimodal_gate.get('full_fivefold_training_authorized')
            is False
        ),
        'multimodal_target_free': (
            multimodal_gate.get('target_arrays_read') is False
        ),
        'multimodal_outcome_free': (
            multimodal_gate.get('outcome_workbooks_read') is False
        ),
        'v2_model_preflight_passed': v2_gate.get('preflight_passed') is True,
        'v2_parameter_count_frozen': int(
            v2_gate.get('total_parameters', -1)
        ) == EXPECTED_V2_PARAMETERS,
        'v2_training_locked': (
            v2_gate.get('model_training_authorized') is False
        ),
    }
    if not all(upstream_checks.values()):
        failed = [name for name, passed in upstream_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 Block 2 upstream checks failed: ' + ', '.join(failed)
        )

    
    if PREFLIGHT_GATE_PATH.exists():
        existing_gate = load_json(PREFLIGHT_GATE_PATH)
        existing_outputs = existing_gate.get('output_sha256', {})
        outputs_stable = all(
            Path(path).exists() and file_sha256(path) == digest
            for path, digest in existing_outputs.items()
        )
        if (
            existing_gate.get('preflight_passed') is True
            and existing_gate.get('code_revision') == CODE_REVISION
            and existing_gate.get('input_sha256') == input_hashes
            and outputs_stable
        ):
            print('\n' + '=' * 80)
            print('TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT ALREADY COMPLETE')
            print('=' * 80)
            print(f'Preflight gate: {PREFLIGHT_GATE_PATH}')
            print('No architecture artifact was rewritten.')
            return

    manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
    required_manifest_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing_columns = sorted(required_manifest_columns - set(manifest_df.columns))
    if missing_columns:
        raise RuntimeError(
            'Multimodal manifest is missing columns: ' + ', '.join(missing_columns)
        )
    required_fold_columns = {
        'subject_id', 'frozen_dev_split', 'development_fold',
        'development_role',
    }
    missing_fold_columns = sorted(required_fold_columns - set(fold_df.columns))
    if missing_fold_columns:
        raise RuntimeError(
            'Fold manifest is missing columns: '
            + ', '.join(missing_fold_columns)
        )
    manifest_df['subject_id'] = manifest_df['subject_id'].astype(str)
    fold_df['subject_id'] = fold_df['subject_id'].astype(str)
    manifest_df = manifest_df.sort_values('subject_id').reset_index(drop=True)
    fold_df = fold_df.sort_values('subject_id').reset_index(drop=True)
    if manifest_df['subject_id'].tolist() != fold_df['subject_id'].tolist():
        raise RuntimeError('Multimodal and fold manifests have different subjects.')
    if 'development_fold' in manifest_df.columns:
        if not np.array_equal(
            manifest_df['development_fold'].astype(int).to_numpy(),
            fold_df['development_fold'].astype(int).to_numpy(),
        ):
            raise RuntimeError('Manifest development-fold assignments disagree.')
    manifest_df['development_fold'] = fold_df['development_fold'].astype(int)
    manifest_df['development_role'] = fold_df['development_role'].astype(str)
    split_counts = manifest_df['dev_split'].value_counts().to_dict()
    fold0_fit_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] != FOLD0)
    )
    fold0_heldout_mask = (
        (manifest_df['dev_split'] == 'train')
        & (manifest_df['development_fold'] == FOLD0)
    )
    manifest_checks = {
        'manifest_has_148_subjects': len(manifest_df) == EXPECTED_SUBJECTS,
        'manifest_subjects_unique': (
            manifest_df['subject_id'].nunique() == EXPECTED_SUBJECTS
        ),
        'split_counts_match': split_counts == EXPECTED_SPLITS,
        'fold0_fit_has_82_subjects': int(fold0_fit_mask.sum()) == EXPECTED_FOLD0_FIT,
        'fold0_heldout_has_20_subjects': (
            int(fold0_heldout_mask.sum()) == EXPECTED_FOLD0_HELDOUT
        ),
        'multimodal_passed_for_all': bool(
            manifest_df[
                'trial8_v2_multimodal_loader_preflight_passed'
            ].map(as_bool).all()
        ),
        'training_locked_for_all': not bool(
            manifest_df['trial8_model_training_authorized'].map(as_bool).any()
        ),
    }
    if not all(manifest_checks.values()):
        failed = [name for name, passed in manifest_checks.items() if not passed]
        raise RuntimeError(
            'Trial8 V2.2 manifest checks failed: ' + ', '.join(failed)
        )

    with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
        schema = {name: np.asarray(data[name]) for name in data.files}
    if any(value.dtype.kind == 'O' for value in schema.values()):
        raise RuntimeError('Verified graph schema contains object arrays.')
    topology_checks = {
        'cross_edge_shape_is_2_by_68': (
            schema['cross_edge_index'].shape == (2, 68)
        ),
        'cross_edge_types_have_68_entries': (
            schema['cross_edge_type'].shape == (68,)
        ),
        'cross_edge_weights_have_68_entries': (
            schema['cross_edge_weight'].shape == (68,)
        ),
        'cross_sources_are_vessels': bool(
            (schema['cross_edge_index'][0] >= 0).all()
            and (schema['cross_edge_index'][0] < NUM_VESSEL_NODES).all()
        ),
        'cross_targets_are_territories': bool(
            (schema['cross_edge_index'][1] >= 0).all()
            and (schema['cross_edge_index'][1] < NUM_TERRITORY_NODES).all()
        ),
        'cross_relation_order_frozen': decode_strings(
            schema['cross_relation_names']
        ) == ['primary_supply', 'communicating_or_proximal_supply'],
    }
    if not all(topology_checks.values()):
        failed = [name for name, passed in topology_checks.items() if not passed]
        raise RuntimeError('Arterial topology checks failed: ' + ', '.join(failed))

    smoke_subject, local_origin = choose_smoke_subject(manifest_df)
    smoke_row = manifest_df.loc[
        manifest_df['subject_id'].eq(smoke_subject)
    ].iloc[0]
    if not (
        str(smoke_row['dev_split']) == 'train'
        and int(smoke_row['development_fold']) != FOLD0
    ):
        raise RuntimeError('Smoke subject is not a fold-0 fitting subject.')
    sample, sample_metadata = load_v22_subject(
        smoke_row, schema, multimodal_gate, local_origin
    )
    target_record = load_smoke_target(smoke_row, local_origin)
    target_read_subjects = {smoke_subject}
    fold0_fit_subjects = set(
        manifest_df.loc[fold0_fit_mask, 'subject_id'].astype(str)
    )
    target_access_checks = {
        'exactly_one_target_read_for_smoke_test': len(target_read_subjects) == 1,
        'target_read_is_fold0_fit_only': target_read_subjects <= fold0_fit_subjects,
        'fold0_heldout_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[fold0_heldout_mask, 'subject_id'].astype(str)
            )
        ),
        'locked_validation_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'val', 'subject_id'
                ].astype(str)
            )
        ),
        'test_targets_not_read': not bool(
            target_read_subjects & set(
                manifest_df.loc[
                    manifest_df['dev_split'] == 'test', 'subject_id'
                ].astype(str)
            )
        ),
        'outcome_workbooks_not_read': True,
    }
    batch_cpu = stack_sample(sample)
    local_target_cpu = torch.from_numpy(
        target_record['local_target'][None, None]
    )
    global_target_cpu = torch.from_numpy(
        target_record['global_target'][None, None]
    )

    set_determinism(SEED)
    model = VascuFateNetV22(schema)
    component_parameters = {
        'preserved_v2_local_path': parameter_count(model.local_model),
        'directed_arterial_projector': parameter_count(
            model.arterial_projector
        ),
        'global_proposal_stream': parameter_count(model.global_stream),
        'global_burden_context_router': parameter_count(model.global_router),
        'proposal_conditioned_local_fusion': parameter_count(
            model.local_fusion
        ),
    }
    total_parameters = parameter_count(model)
    trainable_parameters = parameter_count(model, trainable_only=True)
    new_parameters = total_parameters - component_parameters[
        'preserved_v2_local_path'
    ]
    parameter_checks = {
        'preserved_v2_parameter_count_exact': (
            component_parameters['preserved_v2_local_path']
            == EXPECTED_V2_PARAMETERS
        ),
        'component_parameter_sum_matches_total': (
            sum(component_parameters.values()) == total_parameters
        ),
        'new_parameters_within_2p5m_budget': (
            0 < new_parameters <= MAXIMUM_NEW_PARAMETERS
        ),
        'total_parameters_within_block1_budget': (
            total_parameters <= MAXIMUM_TOTAL_PARAMETERS
        ),
        'all_parameters_trainable_at_initialization': (
            trainable_parameters == total_parameters
        ),
        'direct_graph_to_logit_residual_disabled': (
            model.direct_graph_to_logit_residual is False
        ),
        'expert_routing_disabled': model.expert_routing is False,
    }
    parameter_df = pd.DataFrame([
        {'component': name, 'parameters': count}
        for name, count in component_parameters.items()
    ] + [
        {'component': 'new_v2_2_parameters', 'parameters': new_parameters},
        {'component': 'total', 'parameters': total_parameters},
        {'component': 'trainable', 'parameters': trainable_parameters},
    ])

    if not torch.cuda.is_available():
        raise RuntimeError(
            'Trial8 V2.2 Block 2 requires a CUDA runtime for the L4 memory gate.'
        )
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capacity_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_capacity_gb < GPU_MEMORY_LIMIT_GB:
        raise RuntimeError(
            f'GPU has {gpu_capacity_gb:.2f} GB; expected an L4-class runtime.'
        )
    batch = move_batch(batch_cpu, device)
    local_target = local_target_cpu.to(device)
    global_target = global_target_cpu.to(device)
    model = model.to(device)
    reference_local = deepcopy(model.local_model).to(device)
    model.eval()
    reference_local.eval()

    local_keys = [
        'image', 'region_map', 'territory_features',
        'territory_feature_valid', 'territory_valid',
        'territory_reliability', 'vessel_features',
        'vessel_feature_valid', 'vessel_valid', 'case_features',
        'atlas_enabled', 'vascular_image', 'confidence_features',
        'occlusion_values', 'occlusion_valid', 'clinical_input',
        'vascular_available', 'occlusion_available', 'clinical_available',
    ]
    local_kwargs = {key: batch[key] for key in local_keys}
    common_forces = {
        'force_atlas_dropout': False,
        'force_vascular_dropout': False,
        'force_occlusion_dropout': False,
        'force_clinical_dropout': False,
    }
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        reference_output = reference_local(**local_kwargs, **common_forces)
        neutral_output = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    neutral_differences = {
        name: tensor_maximum_difference(
            neutral_output[name], reference_output[name]
        ) for name in [
            'logits', 'auxiliary_half_logits',
            'auxiliary_quarter_logits', 'territory_node_logits',
            'global_graph_context', 'fused_global_context',
        ]
    }
    neutral_checks = {
        'initial_local_logits_match_v2_exactly': (
            neutral_differences['logits'] <= 1e-6
        ),
        'initial_half_logits_match_v2_exactly': (
            neutral_differences['auxiliary_half_logits'] <= 1e-6
        ),
        'initial_quarter_logits_match_v2_exactly': (
            neutral_differences['auxiliary_quarter_logits'] <= 1e-6
        ),
        'initial_v2_node_logits_match_exactly': (
            neutral_differences['territory_node_logits'] <= 1e-6
        ),
        'initial_v2_graph_context_matches_exactly': (
            neutral_differences['global_graph_context'] <= 1e-6
        ),
        'initial_v2_fused_context_matches_exactly': (
            neutral_differences['fused_global_context'] <= 1e-6
        ),
        'proposal_fusion_gate_exactly_zero': bool(
            neutral_output['proposal_fusion_gate'].abs().max().item() == 0.0
        ),
        'global_context_film_exactly_zero_initialized': bool(
            model.global_stream.context_film.project[-1].weight.abs().max().item()
            == 0.0
            and model.global_stream.context_film.project[-1].bias.abs().max().item()
            == 0.0
        ),
    }
    del reference_local, reference_output
    torch.cuda.empty_cache()

    expected_shapes = {
        'logits': (1, 1, *LOCAL_PATCH_SIZE),
        'auxiliary_half_logits': (1, 1, 64, 64, 32),
        'auxiliary_quarter_logits': (1, 1, 32, 32, 16),
        'territory_node_logits': (1, 30),
        'global_graph_context': (1, 64),
        'global_proposal_logits': (1, 1, *GLOBAL_INPUT_SHAPE),
        'global_proposal_local': (1, 1, *LOCAL_PATCH_SIZE),
        'global_feature_local': (1, GLOBAL_CHANNELS[0], *LOCAL_PATCH_SIZE),
        'arterial_territory_logits': (1, 30),
        'arterial_territory_valid': (1, 30),
        'arterial_global_context': (1, 64),
        'arterial_global_map': (1, *GLOBAL_INPUT_SHAPE),
        'lesion_presence_logit': (1,),
        'log_volume_prediction': (1,),
        'global_burden_context': (1, 64),
        'proposal_fusion_gate': (1, 15),
    }
    shape_rows = []
    for name, expected_shape in expected_shapes.items():
        observed = tuple(neutral_output[name].shape)
        shape_rows.append({
            'output': name,
            'observed_shape': str(observed),
            'expected_shape': str(expected_shape),
            'shape_matches': observed == expected_shape,
            'finite': bool(
                torch.isfinite(neutral_output[name]).all()
                if neutral_output[name].dtype.is_floating_point else True
            ),
        })
    shape_df = pd.DataFrame(shape_rows)
    output_checks = {
        'all_output_shapes_match': bool(shape_df['shape_matches'].all()),
        'all_normal_outputs_finite': all_tensor_outputs_finite(neutral_output),
        'global_proposal_is_nonconstant': float(
            neutral_output['global_proposal_logits'].std().cpu()
        ) > 1e-6,
        'arterial_projection_has_valid_nodes': bool(
            neutral_output['arterial_territory_valid'].any()
        ) if bool(batch['atlas_enabled'].item()) else True,
    }

    
    coordinate_fields = []
    for axis, size in enumerate(GLOBAL_INPUT_SHAPE):
        vector = 2.0 * (
            torch.arange(size, device=device, dtype=torch.float32) + 0.5
        ) / float(size) - 1.0
        shape = [1, 1, 1]
        shape[axis] = size
        field = vector.view(*shape).expand(*GLOBAL_INPUT_SHAPE)
        coordinate_fields.append(field)
    coordinate_tensor = torch.stack(coordinate_fields, dim=0)[None]
    sampled_coordinates = sample_global_to_local(
        coordinate_tensor, batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    expected_coordinate_fields = []
    for axis in range(3):
        vector = 2.0 * (
            batch['local_origin_in_canvas'][:, axis:axis + 1].float()
            + torch.arange(
                LOCAL_PATCH_SIZE[axis], device=device, dtype=torch.float32
            )[None]
            + 0.5
        ) / float(GLOBAL_CANVAS_SHAPE[axis]) - 1.0
        
        
        
        
        center_limit = 1.0 / float(GLOBAL_INPUT_SHAPE[axis])
        vector = vector.clamp(-1.0 + center_limit, 1.0 - center_limit)
        shape = [1, 1, 1, 1]
        shape[axis + 1] = LOCAL_PATCH_SIZE[axis]
        expected_coordinate_fields.append(
            vector.view(*shape).expand(1, *LOCAL_PATCH_SIZE)
        )
    expected_coordinates = torch.stack(expected_coordinate_fields, dim=1)
    alignment_error = tensor_maximum_difference(
        sampled_coordinates, expected_coordinates
    )
    alignment_checks = {
        'global_to_local_grid_axis_order_correct': alignment_error <= 2e-5,
        'local_origin_has_three_coordinates': (
            tuple(batch['local_origin_in_canvas'].shape) == (1, 3)
        ),
        'local_target_is_nonempty': target_record['local_support_voxels'] > 0,
        'global_maxpool_target_is_nonempty': (
            target_record['global_support_voxels'] > 0
        ),
        'global_maxpool_never_erases_target_support': (
            target_record['global_support_voxels'] > 0
        ),
    }

    with torch.no_grad():
        changed_global_batch = dict(batch)
        changed_global_batch['global_image'] = torch.roll(
            batch['global_image'], shifts=7, dims=2
        )
        changed_global = model(
            **changed_global_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        missing_batch = dict(batch)
        missing_batch['atlas_enabled'] = torch.zeros_like(
            batch['atlas_enabled']
        )
        missing_batch['territory_valid'] = torch.zeros_like(
            batch['territory_valid']
        )
        missing_batch['vessel_valid'] = torch.zeros_like(batch['vessel_valid'])
        missing_batch['global_region_map'] = torch.zeros_like(
            batch['global_region_map']
        )
        missing_batch['clinical_available'] = torch.zeros_like(
            batch['clinical_available']
        )
        missing_batch['occlusion_available'] = torch.zeros_like(
            batch['occlusion_available']
        )
        missing_output = model(
            **missing_batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
    robustness_checks = {
        'global_proposal_responds_to_global_image': (
            tensor_maximum_difference(
                neutral_output['global_proposal_logits'],
                changed_global['global_proposal_logits'],
            ) > 1e-6
        ),
        'missing_branch_outputs_are_finite': all_tensor_outputs_finite(
            missing_output
        ),
        'missing_atlas_has_no_arterial_projection': not bool(
            missing_output['arterial_territory_valid'].any()
        ),
        'missing_atlas_has_zero_arterial_map': bool(
            missing_output['arterial_global_map'].abs().max().item() == 0.0
        ),
    }
    del changed_global, missing_output

    
    model.activate_new_gates_for_gradient_test()
    model.eval()
    with torch.no_grad():
        activated = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=False,
        )
        no_global = model(
            **batch, **common_forces,
            force_global_dropout=True, force_arterial_dropout=False,
        )
        no_arterial = model(
            **batch, **common_forces,
            force_global_dropout=False, force_arterial_dropout=True,
        )
        no_clinical = model(
            **batch,
            force_atlas_dropout=False,
            force_vascular_dropout=False,
            force_occlusion_dropout=False,
            force_clinical_dropout=True,
            force_global_dropout=False,
            force_arterial_dropout=False,
        )
    ablation_differences = {
        'no_global_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_global['logits']
        ),
        'no_arterial_global_proposal_difference': tensor_maximum_difference(
            activated['global_proposal_logits'],
            no_arterial['global_proposal_logits'],
        ),
        'no_arterial_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_arterial['logits']
        ),
        'no_clinical_presence_difference': tensor_maximum_difference(
            activated['lesion_presence_logit'],
            no_clinical['lesion_presence_logit'],
        ),
        'no_clinical_local_logit_difference': tensor_maximum_difference(
            activated['logits'], no_clinical['logits']
        ),
    }
    ablation_checks = {
        'activated_proposal_gate_is_nonzero': bool(
            activated['proposal_fusion_gate'].abs().max().item() > 0
        ),
        'global_path_ablation_is_responsive': (
            ablation_differences['no_global_local_logit_difference'] > 1e-7
        ),
        'arterial_path_changes_global_proposal': (
            ablation_differences[
                'no_arterial_global_proposal_difference'
            ] > 1e-7
        ),
        'arterial_path_reaches_local_features': (
            ablation_differences['no_arterial_local_logit_difference'] > 1e-8
        ),
        'clinical_path_changes_presence_burden_context': (
            ablation_differences['no_clinical_presence_difference'] > 1e-8
        ),
        'clinical_path_reaches_local_only_through_bounded_context': (
            ablation_differences['no_clinical_local_logit_difference'] > 1e-9
        ),
    }
    ablation_df = pd.DataFrame([
        {'diagnostic': name, 'maximum_absolute_difference': value}
        for name, value in ablation_differences.items()
    ])
    del no_global, no_arterial, no_clinical

    model.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats()
    gradient_output = model(
        **batch, **common_forces,
        force_global_dropout=False, force_arterial_dropout=False,
    )
    local_bce = F.binary_cross_entropy_with_logits(
        gradient_output['logits'], local_target
    )
    local_dice = soft_dice_loss(gradient_output['logits'], local_target)
    global_bce = F.binary_cross_entropy_with_logits(
        gradient_output['global_proposal_logits'], global_target
    )
    global_dice = soft_dice_loss(
        gradient_output['global_proposal_logits'], global_target
    )
    presence_target = torch.ones_like(
        gradient_output['lesion_presence_logit']
    )
    presence_loss = F.binary_cross_entropy_with_logits(
        gradient_output['lesion_presence_logit'], presence_target
    )
    target_volume = global_target.sum(dim=(1, 2, 3, 4))
    log_volume_target = torch.log1p(target_volume)
    volume_loss = F.smooth_l1_loss(
        gradient_output['log_volume_prediction'], log_volume_target
    )
    territory_target, territory_supervision_valid = (
        territory_target_from_global(
            global_target, batch['global_region_map']
        )
    )
    arterial_supervision_valid = (
        territory_supervision_valid
        & gradient_output['arterial_territory_valid']
    )
    if bool(arterial_supervision_valid.any()):
        arterial_loss = F.binary_cross_entropy_with_logits(
            gradient_output['arterial_territory_logits'][
                arterial_supervision_valid
            ],
            territory_target[arterial_supervision_valid],
        )
    else:
        arterial_loss = gradient_output[
            'arterial_territory_logits'
        ].sum() * 0.0
    v2_node_valid = batch['territory_valid'].bool()
    if bool(v2_node_valid.any()):
        v2_node_loss = F.binary_cross_entropy_with_logits(
            gradient_output['territory_node_logits'][v2_node_valid],
            territory_target[v2_node_valid],
        )
    else:
        v2_node_loss = gradient_output['territory_node_logits'].sum() * 0.0
    smoke_loss = (
        0.35 * local_bce + 0.65 * local_dice
        + 0.25 * global_bce + 0.75 * global_dice
        + 0.10 * presence_loss + 0.05 * volume_loss
        + 0.10 * arterial_loss + 0.05 * v2_node_loss
    )
    smoke_loss.backward()
    peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9
    gradient_modules = {
        'preserved_local_image_backbone': model.local_model.stem,
        'preserved_local_graph_encoder': model.local_model.graph_encoder,
        'preserved_local_segmentation_head': (
            model.local_model.segmentation_head
        ),
        'global_proposal_stream': model.global_stream,
        'directed_arterial_projector': model.arterial_projector,
        'proposal_conditioned_local_fusion': model.local_fusion,
        'global_burden_context_router': model.global_router,
        'clinical_context_encoder': model.global_router.clinical_encoder,
        'global_context_film': model.global_stream.context_film,
    }
    gradient_rows = []
    gradient_checks = {}
    for name, module in gradient_modules.items():
        norm, observed, finite = gradient_norm(module)
        gradient_rows.append({
            'component': name, 'gradient_norm': norm,
            'gradient_observed': observed, 'gradient_finite': finite,
        })
        gradient_checks[f'{name}_has_finite_nonzero_gradient'] = (
            observed and finite and norm > 0.0
        )
    gradient_df = pd.DataFrame(gradient_rows)
    loss_checks = {
        'smoke_loss_is_finite': bool(torch.isfinite(smoke_loss)),
        'local_loss_is_finite': bool(
            torch.isfinite(local_bce) and torch.isfinite(local_dice)
        ),
        'global_proposal_loss_is_finite': bool(
            torch.isfinite(global_bce) and torch.isfinite(global_dice)
        ),
        'presence_loss_is_finite': bool(torch.isfinite(presence_loss)),
        'volume_loss_is_finite': bool(torch.isfinite(volume_loss)),
        'arterial_auxiliary_loss_is_finite': bool(
            torch.isfinite(arterial_loss)
        ),
        'peak_memory_below_l4_safety_limit': (
            peak_memory_gb < GPU_MEMORY_LIMIT_GB
        ),
    }

    architecture_record = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preserved_v2_local_path': {
            'parameters': component_parameters['preserved_v2_local_path'],
            'checkpoint_warm_start': False,
            'initial_output_equivalence_required': True,
        },
        'global_stream': {
            'canvas_shape': list(GLOBAL_CANVAS_SHAPE),
            'input_shape': list(GLOBAL_INPUT_SHAPE),
            'input_channels': 17,
            'channels': list(GLOBAL_CHANNELS),
            'proposal_output': 'coarse full-brain occupancy logits',
        },
        'arterial_projector': {
            'vessel_nodes': NUM_VESSEL_NODES,
            'territory_nodes': NUM_TERRITORY_NODES,
            'directed_edges': 68,
            'relation_priors': {
                'primary_supply': 1.0,
                'communicating_or_proximal_supply': 0.5,
            },
            'spatialization': 'registered territory map only',
            'direct_final_logit_residual': False,
        },
        'clinical_role': {
            'presence_and_burden': True,
            'bounded_global_channel_context': True,
            'direct_spatial_localization': False,
            'direct_final_logit_residual': False,
            'post_imaging_treatment_fields': False,
            'post_treatment_outcomes': False,
        },
        'local_fusion': {
            'type': 'proposal-conditioned input-feature residual',
            'zero_initialized_gate': True,
            'bound': FUSION_BOUND,
            'direct_logit_residual': False,
        },
        'parameters': {
            **component_parameters,
            'new_v2_2_parameters': new_parameters,
            'total': total_parameters,
            'trainable': trainable_parameters,
            'maximum_new_parameters': MAXIMUM_NEW_PARAMETERS,
        },
        'smoke_subject': smoke_subject,
        'smoke_subject_local_origin': list(local_origin),
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_training_authorized': False,
        'loader_and_pilot_implementation_authorized': False,
    }

    all_checks = {
        **upstream_checks,
        **manifest_checks,
        **topology_checks,
        **target_access_checks,
        **parameter_checks,
        **neutral_checks,
        **output_checks,
        **alignment_checks,
        **robustness_checks,
        **ablation_checks,
        **gradient_checks,
        **loss_checks,
        'study_model_constructed_for_preflight_only': True,
        'optimizer_steps_are_zero': True,
        'checkpoint_not_written': True,
        'pilot_training_remains_locked': True,
        'full_fivefold_training_remains_locked': True,
        'outcome_workbooks_not_read': True,
    }
    preflight_passed = bool(all(bool(value) for value in all_checks.values()))
    architecture_record['loader_and_pilot_implementation_authorized'] = (
        preflight_passed
    )
    atomic_write_json(architecture_record, ARCHITECTURE_PATH)
    atomic_write_csv(parameter_df, PARAMETER_AUDIT_PATH)
    atomic_write_csv(shape_df, OUTPUT_SHAPE_AUDIT_PATH)
    atomic_write_csv(gradient_df, GRADIENT_AUDIT_PATH)
    atomic_write_csv(ablation_df, ABLATION_AUDIT_PATH)
    audit_df = pd.DataFrame([
        {'check': name, 'passed': bool(passed)}
        for name, passed in all_checks.items()
    ])
    atomic_write_csv(audit_df, PREFLIGHT_AUDIT_PATH)
    output_paths = [
        ARCHITECTURE_PATH, PARAMETER_AUDIT_PATH, OUTPUT_SHAPE_AUDIT_PATH,
        GRADIENT_AUDIT_PATH, ABLATION_AUDIT_PATH, PREFLIGHT_AUDIT_PATH,
    ]
    output_hashes = {str(path): file_sha256(path) for path in output_paths}
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'model_version': MODEL_VERSION,
        'preflight_version': PREFLIGHT_VERSION,
        'code_revision': CODE_REVISION,
        'preflight_passed': preflight_passed,
        'checks': all_checks,
        'input_sha256': input_hashes,
        'output_sha256': output_hashes,
        'architecture_path': str(ARCHITECTURE_PATH),
        'architecture_sha256': output_hashes[str(ARCHITECTURE_PATH)],
        'parameter_audit_path': str(PARAMETER_AUDIT_PATH),
        'output_shape_audit_path': str(OUTPUT_SHAPE_AUDIT_PATH),
        'gradient_audit_path': str(GRADIENT_AUDIT_PATH),
        'ablation_audit_path': str(ABLATION_AUDIT_PATH),
        'preflight_audit_path': str(PREFLIGHT_AUDIT_PATH),
        'smoke_subject': smoke_subject,
        'target_arrays_read_subjects': [smoke_subject],
        'target_arrays_read_splits': ['train_fold0_fit'],
        'fold0_heldout_targets_read': False,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'outcome_workbooks_read': False,
        'total_parameters': total_parameters,
        'new_parameters': new_parameters,
        'peak_gpu_memory_gb': peak_memory_gb,
        'study_model_constructed': True,
        'training_performed': False,
        'optimizer_steps': 0,
        'checkpoint_written': False,
        'pilot_loader_implementation_authorized': preflight_passed,
        'pilot_training_authorized': False,
        'full_fivefold_training_authorized': False,
        'next_block': (
            'build the isolated fold-0 V2.2 pilot loader/training cell with '
            'component recall, proposal coverage, branch reliance, lesion-size '
            'strata, calibration, and early-stopping diagnostics'
        ),
    }
    atomic_write_json(gate, PREFLIGHT_GATE_PATH)

    print('\nTrial8 V2.2 Block 2 upstream gates passed.')
    print(f'Smoke subject:                      {smoke_subject} (fold-0 fit)')
    print(f'Local origin in global canvas:      {local_origin}')
    print(f'GPU:                                {gpu_name}')
    print(f'GPU capacity GB:                    {gpu_capacity_gb:.2f}')
    print(f'Fold-0 held-out targets read:       False')
    print(f'Locked validation targets read:     False')
    print(f'Test targets read:                  False')
    print(f'Optimizer steps:                    0')
    print('\nParameter count:')
    display(parameter_df)
    print('\nOutput shapes:')
    display(shape_df)
    print('\nControlled ablation response:')
    display(ablation_df)
    print('\nGradient norms:')
    display(gradient_df)
    print('\nModel preflight checks:')
    display(audit_df)
    if not preflight_passed:
        failed = [name for name, passed in all_checks.items() if not passed]
        raise RuntimeError(
            'TRIAL8 V2.2 MODEL CONSTRUCTION PREFLIGHT FAILED. Do not train. '
            'Failed checks: ' + ', '.join(failed)
        )
    print('\n' + '=' * 80)
    print('TRIAL8 V2.2 GLOBAL-TO-LOCAL MODEL CONSTRUCTION PREFLIGHT PASSED')
    print('=' * 80)
    print(f'Preserved V2 parameters:            {EXPECTED_V2_PARAMETERS:,}')
    print(f'New V2.2 parameters:                {new_parameters:,}')
    print(f'Total parameters:                   {total_parameters:,}')
    print(f'Peak GPU memory GB:                 {peak_memory_gb:.3f}')
    print('Initial local path equals V2:       True')
    print('Global full-brain proposal:         verified')
    print('Directed arterial projection:       verified (68 edges)')
    print('Clinical role:                      presence/burden/global context')
    print('Direct graph-to-logit residual:     False')
    print('Expert routing:                     False')
    print('Checkpoint warm start:              False')
    print('Training performed:                 False')
    print('Pilot training authorized:          False')
    print(f'Pilot implementation allowed:       {preflight_passed}')
    print(f'Architecture:                       {ARCHITECTURE_PATH}')
    print(f'Preflight gate:                     {PREFLIGHT_GATE_PATH}')
    print(
        '\nNext: build the isolated fold-0 V2.2 pilot with proposal/component '
        'coverage and branch-reliance diagnostics; do not launch five folds.'
    )


main()

















import gc
import random
import time
from collections import defaultdict

scipy = ensure_package('scipy')
from scipy import ndimage






PILOT_CODE_REVISION = 'r2_strict_numeric_best_independent_patience'
REPLICATION_SERIES_VERSION = 'v2_2_r5_final_fivefold_strictbest_v2'
AUTHORIZED_REPLICATION_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_DEVELOPMENT_FOLD_COUNTS = {0: 20, 1: 20, 2: 20, 3: 21, 4: 21}
EXPECTED_V22_PARAMETERS = 16_870_679
CHECKPOINT_SELECTION_MINIMUM_DELTA = 0.0
FINAL_PROTOCOL_RETRAINS_FOLD0 = True
FINAL_PROTOCOL_ROLE = (
    'development_fivefold_estimation_not_locked_validation_or_test'
)

REPLICATION_SERIES_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / REPLICATION_SERIES_VERSION
)


def replication_result_is_complete(fold):
    root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
    resolved_root = root.resolve()
    gate_path = root / 'final_fold_result_gate.json'
    best_path = root / 'best_checkpoint.pt'
    if not gate_path.exists() or not best_path.exists():
        return False
    try:
        gate = load_json(gate_path)
        required_artifacts = {
            'validation_summary', 'case_metrics', 'component_metrics',
            'component_tier_metrics', 'local_threshold_summary',
            'local_threshold_case_metrics', 'proposal_threshold_summary',
            'proposal_threshold_case_metrics', 'branch_reliance',
            'branch_reliance_history', 'fit_target_access_audit',
            'fit_component_audit', 'epoch1_schedule_audit',
            'epoch1_negative_skip_audit',
        }
        artifact_manifest = gate.get('best_epoch_artifact_manifest', {})
        if not isinstance(artifact_manifest, dict):
            return False
        if set(artifact_manifest) != required_artifacts:
            return False
        for record in artifact_manifest.values():
            if not isinstance(record, dict):
                return False
            artifact_path = Path(str(record.get('path', ''))).resolve()
            artifact_path.relative_to(resolved_root)
            if (
                not artifact_path.is_file()
                or record.get('sha256') != file_sha256(artifact_path)
            ):
                return False
        history_path = Path(str(gate.get('history_path', '')))
        config_path = Path(str(gate.get('pilot_config_path', '')))
        if history_path != root / 'training_history.csv':
            return False
        if config_path != root / 'pilot_config.json':
            return False
        if (
            not history_path.is_file()
            or gate.get('history_sha256') != file_sha256(history_path)
            or not config_path.is_file()
            or gate.get('pilot_config_sha256') != file_sha256(config_path)
        ):
            return False
        strict_best_checks = gate.get('strict_best_checks', {})
        best_oof_checks = gate.get(
            'best_oof_sufficient_statistics_checks', {}
        )
        target_access_checks = gate.get('target_access_checks', {})
        final_stop_checks = gate.get('final_stop_checks', {})
        for checks in [
            strict_best_checks, best_oof_checks, target_access_checks,
            final_stop_checks,
        ]:
            if not isinstance(checks, dict) or not checks:
                return False
            if not all(value is True for value in checks.values()):
                return False
        return bool(
            gate.get('pilot_complete') is True
            and gate.get('final_fold_complete') is True
            and gate.get('replication_complete') is True
            and gate.get('development_fold') == int(fold)
            and gate.get('heldout_subjects')
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[int(fold)]
            and gate.get('parameter_count') == EXPECTED_V22_PARAMETERS
            and isinstance(gate.get('fit_subject_hash'), str)
            and len(gate.get('fit_subject_hash')) == 64
            and isinstance(gate.get('heldout_subject_hash'), str)
            and len(gate.get('heldout_subject_hash')) == 64
            and gate.get('pilot_version')
            == f'v2_2_r5_final_strictbest_fold{int(fold)}_r2'
            and gate.get('code_revision') == PILOT_CODE_REVISION
            and gate.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and gate.get('fresh_start_from_initialization') is True
            and gate.get('parent_checkpoint_migration_allowed') is False
            and gate.get('strict_numeric_best_verified') is True
            and gate.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(gate.get(
                'checkpoint_selection_minimum_delta', float('nan')
            )) == CHECKPOINT_SELECTION_MINIMUM_DELTA
            and gate.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and gate.get(
                'pooled_oof_threshold_calibration_sufficient_statistics_written'
            ) is True
            and gate.get(
                'fold_selected_threshold_is_not_final_deployment_threshold'
            ) is True
            and gate.get('final_protocol_role') == FINAL_PROTOCOL_ROLE
            and gate.get('final_protocol_retrains_fold0') is True
            and gate.get(
                'fold0_architecture_development_bias_acknowledged'
            ) is True
            and gate.get('architecture_changed_from_r5') is False
            and gate.get('optimization_loss_sampling_changed_from_r5') is False
            and gate.get('checkpoint_selection_semantics_changed_from_r5')
            is True
            and gate.get('r5_source_checkpoint_unchanged') is True
            and gate.get('r5_source_checkpoint_loaded_for_initialization')
            is False
            and gate.get('active_fold_training_authorized') is True
            and gate.get('development_fivefold_replication_authorized') is True
            and gate.get('manual_review_required_before_next_fold') is False
            and gate.get('locked_validation_targets_read') is False
            and gate.get('test_targets_read') is False
            and gate.get('best_checkpoint_path') == str(best_path)
            and gate.get('best_checkpoint_sha256') == file_sha256(best_path)
        )
    except (KeyError, TypeError, ValueError, OSError, json.JSONDecodeError):
        return False


COMPLETED_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if replication_result_is_complete(fold)
)
REMAINING_REPLICATION_FOLDS = tuple(
    fold for fold in AUTHORIZED_REPLICATION_FOLDS
    if fold not in COMPLETED_REPLICATION_FOLDS
)
ALL_REPLICATION_FOLDS_COMPLETE = not REMAINING_REPLICATION_FOLDS
REPLICATION_FOLD = (
    REMAINING_REPLICATION_FOLDS[0]
    if REMAINING_REPLICATION_FOLDS
    else AUTHORIZED_REPLICATION_FOLDS[-1]
)
RUNTIME_FOLD = REPLICATION_FOLD
PILOT_VERSION = f'v2_2_r5_final_strictbest_fold{REPLICATION_FOLD}_r2'
FRESH_START_FROM_INITIALIZATION = True
PARENT_CHECKPOINT_MIGRATION_ALLOWED = False
PILOT_SEED = 20261002
MAX_EPOCHS = 40
MINIMUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MINIMUM_DELTA = 0.002
STRICT_BEST_SCORE_ATOL = 1.0e-12
POSITIVE_PATCHES = 112
TOPOLOGY_NEGATIVE_PATCHES = 28
BRAIN_NEGATIVE_PATCHES = 28
ACCUMULATION_STEPS = 4
BASE_LR = 1.0e-4
NEW_PATH_LR = 2.0e-4
GATE_LR = 4.0e-4
WEIGHT_DECAY = 1.0e-4
GRADIENT_CLIP_NORM = 2.0
EMA_MAX_DECAY = 0.995
VALIDATION_OVERLAP = 0.50
LOCAL_THRESHOLDS = np.round(np.arange(0.05, 0.81, 0.05), 2)
PROPOSAL_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
COMPONENT_TIERS = [
    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small', 'medium', 'large'
]

V22_PREFLIGHT_GATE_PATH = PREFLIGHT_GATE_PATH
V22_ARCHITECTURE_PATH = ARCHITECTURE_PATH
FIVEFOLD_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'training_preflight'
    / 'trial8_vascufate_net_v1'
    / 'v2_fivefold_softsubvoxel_sampler_loss_auditfix'
)
FIVEFOLD_GATE_PATH = FIVEFOLD_ROOT / 'fivefold_loader_loss_sampler_gate.json'

AUTHORIZATION_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / 'v1_r5_metric_reconciliation_fivefold_authorization'
)
AUTHORIZATION_GATE_PATH = (
    AUTHORIZATION_ROOT / 'r5_fivefold_evaluation_authorization_gate.json'
)
R5_SOURCE_ROOT = (
    PROJECT_ROOT / 'checkpoints' / MODEL_VERSION
    / 'v2_2_fold0_global_local_component_diagnostic_r5'
)
R5_SOURCE_CONFIG_PATH = R5_SOURCE_ROOT / 'pilot_config.json'
R5_SOURCE_RESULT_GATE_PATH = R5_SOURCE_ROOT / 'pilot_result_gate.json'
R5_SOURCE_BEST_CHECKPOINT_PATH = R5_SOURCE_ROOT / 'best_checkpoint.pt'

PILOT_ROOT = REPLICATION_SERIES_ROOT / f'fold_{REPLICATION_FOLD}'
PILOT_VALIDATION_ROOT = PILOT_ROOT / 'validation'
PILOT_DIAGNOSTIC_ROOT = PILOT_ROOT / 'diagnostics'
PILOT_CONFIG_PATH = PILOT_ROOT / 'pilot_config.json'
PILOT_START_GATE_PATH = PILOT_ROOT / 'pilot_start_gate.json'
PILOT_HISTORY_PATH = PILOT_ROOT / 'training_history.csv'
PILOT_LAST_CHECKPOINT_PATH = PILOT_ROOT / 'last_checkpoint.pt'
PILOT_BEST_CHECKPOINT_PATH = PILOT_ROOT / 'best_checkpoint.pt'
PILOT_STATE_PATH = PILOT_ROOT / 'pilot_state.json'
PILOT_RESULT_GATE_PATH = PILOT_ROOT / 'final_fold_result_gate.json'
FINAL_SERIES_STATUS_PATH = (
    REPLICATION_SERIES_ROOT / 'final_fivefold_status.json'
)
PILOT_TARGET_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_target_access_audit.csv'
)
PILOT_COMPONENT_AUDIT_PATH = (
    PILOT_ROOT / f'fold{REPLICATION_FOLD}_fit_component_audit.csv'
)
PILOT_SCHEDULE_AUDIT_PATH = PILOT_ROOT / 'epoch1_schedule_audit.csv'
PILOT_NEGATIVE_SKIP_AUDIT_PATH = (
    PILOT_ROOT / 'epoch1_negative_candidate_skip_audit.csv'
)
PILOT_BRANCH_HISTORY_PATH = PILOT_ROOT / 'branch_reliance_history.csv'






if not ALL_REPLICATION_FOLDS_COMPLETE and PILOT_RESULT_GATE_PATH.exists():
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
        'the final artifact-integrity contract. Refusing to skip, resume, or '
        'overwrite this fold. Preserve the directory and diagnose the gate '
        f'before continuing: {PILOT_RESULT_GATE_PATH}'
    )

PILOT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIAGNOSTIC_ROOT.mkdir(parents=True, exist_ok=True)


def write_final_series_status():
    fold_records = []
    completed_folds = []
    for fold in AUTHORIZED_REPLICATION_FOLDS:
        root = REPLICATION_SERIES_ROOT / f'fold_{int(fold)}'
        gate_path = root / 'final_fold_result_gate.json'
        checkpoint_path = root / 'best_checkpoint.pt'
        if not replication_result_is_complete(fold):
            continue
        gate = load_json(gate_path)
        completed_folds.append(int(fold))
        fold_records.append({
            'development_fold': int(fold),
            'heldout_subjects': int(gate['heldout_subjects']),
            'parameter_count': int(gate['parameter_count']),
            'fit_subject_hash': str(gate['fit_subject_hash']),
            'heldout_subject_hash': str(gate['heldout_subject_hash']),
            'best_epoch': int(gate['best_epoch']),
            'best_score': float(gate['best_score']),
            'positive_mean_dice': float(
                gate['best_validation_summary']['positive_mean_dice']
            ),
            'positive_global_dice': float(
                gate['best_validation_summary']['positive_global_dice']
            ),
            'lesion_f1': float(
                gate['best_validation_summary']['lesion_f1']
            ),
            'strict_numeric_best_verified': bool(
                gate.get('strict_numeric_best_verified') is True
            ),
            'locked_validation_targets_read': bool(
                gate.get('locked_validation_targets_read') is True
            ),
            'test_targets_read': bool(gate.get('test_targets_read') is True),
            'result_gate_path': str(gate_path),
            'result_gate_sha256': file_sha256(gate_path),
            'best_checkpoint_path': str(checkpoint_path),
            'best_checkpoint_sha256': file_sha256(checkpoint_path),
        })
    status = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'authorized_folds': list(AUTHORIZED_REPLICATION_FOLDS),
        'completed_folds': completed_folds,
        'remaining_folds': [
            fold for fold in AUTHORIZED_REPLICATION_FOLDS
            if fold not in completed_folds
        ],
        'series_complete': (
            tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
        ),
        'completed_heldout_subjects': int(sum(
            record['heldout_subjects'] for record in fold_records
        )),
        'all_strict_numeric_best_verified': bool(
            fold_records
            and all(
                record['strict_numeric_best_verified']
                for record in fold_records
            )
        ),
        'locked_validation_targets_read': bool(any(
            record['locked_validation_targets_read']
            for record in fold_records
        )),
        'test_targets_read': bool(any(
            record['test_targets_read'] for record in fold_records
        )),
        'fold_records': fold_records,
        'next_action': (
            'Build pooled out-of-fold aggregation and freeze one threshold.'
            if tuple(completed_folds) == AUTHORIZED_REPLICATION_FOLDS
            else 'Rerun the unchanged block to train the next missing fold.'
        ),
    }
    status['series_integrity_checks'] = {
        'completed_fold_ids_are_unique': (
            len(completed_folds) == len(set(completed_folds))
        ),
        'all_completed_folds_use_expected_parameter_count': all(
            record['parameter_count'] == EXPECTED_V22_PARAMETERS
            for record in fold_records
        ),
        'all_completed_folds_use_strict_numeric_best': all(
            record['strict_numeric_best_verified'] for record in fold_records
        ),
        'completed_fold_partitions_match_frozen_manifest': all(
            record['fit_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    != record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subject_hash'] == pilot_subject_hash(
                development_df.loc[
                    development_df['development_fold']
                    == record['development_fold'],
                    'subject_id',
                ]
            )
            and record['heldout_subjects']
            == EXPECTED_DEVELOPMENT_FOLD_COUNTS[
                record['development_fold']
            ]
            for record in fold_records
        ),
        'locked_validation_targets_remain_sealed': not status[
            'locked_validation_targets_read'
        ],
        'test_targets_remain_sealed': not status['test_targets_read'],
        'complete_series_has_all_102_development_oof_subjects': (
            not status['series_complete']
            or status['completed_heldout_subjects'] == 102
        ),
    }
    if not all(status['series_integrity_checks'].values()):
        failed = [
            name for name, passed in status['series_integrity_checks'].items()
            if not passed
        ]
        raise RuntimeError(
            'Completed five-fold series integrity audit failed: '
            + ', '.join(failed)
        )
    atomic_write_json(status, FINAL_SERIES_STATUS_PATH)
    return status


def pilot_atomic_torch_save(record, path):
    temporary = Path(str(path) + '.tmp')
    torch.save(record, temporary)
    os.replace(temporary, path)


def pilot_subject_hash(subjects):
    payload = '\n'.join(sorted(str(v) for v in subjects)).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_capture_rng_state():
    state = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    return state


def pilot_restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])

    
    
    
    
    
    def cpu_byte_rng_tensor(value):
        if isinstance(value, torch.Tensor):
            return value.detach().to(
                device='cpu', dtype=torch.uint8
            ).contiguous()
        return torch.as_tensor(
            value, dtype=torch.uint8, device='cpu'
        ).contiguous()

    torch.set_rng_state(cpu_byte_rng_tensor(state['torch_cpu']))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        cuda_states = [
            cpu_byte_rng_tensor(value) for value in state['torch_cuda']
        ]
        if len(cuda_states) != torch.cuda.device_count():
            raise RuntimeError(
                'Checkpoint CUDA RNG-state count does not match the active '
                f'runtime: checkpoint={len(cuda_states)}, '
                f'runtime={torch.cuda.device_count()}.'
            )
        torch.cuda.set_rng_state_all(cuda_states)


pilot_required_paths = [
    V22_PREFLIGHT_GATE_PATH, V22_ARCHITECTURE_PATH, BLOCK1_GATE_PATH,
    MULTIMODAL_GATE_PATH, MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH,
    GRAPH_SCHEMA_PATH, FIVEFOLD_GATE_PATH, AUTHORIZATION_GATE_PATH,
    R5_SOURCE_CONFIG_PATH, R5_SOURCE_RESULT_GATE_PATH,
    R5_SOURCE_BEST_CHECKPOINT_PATH,
]
pilot_missing_paths = [str(p) for p in pilot_required_paths if not p.exists()]
if pilot_missing_paths:
    raise FileNotFoundError(
        'Trial8 V2.2 R5 replication prerequisites are missing:\n'
        + '\n'.join(pilot_missing_paths)
    )

v22_gate = load_json(V22_PREFLIGHT_GATE_PATH)
block1_gate = load_json(BLOCK1_GATE_PATH)
multimodal_gate = load_json(MULTIMODAL_GATE_PATH)
fivefold_gate = load_json(FIVEFOLD_GATE_PATH)
authorization_gate = load_json(AUTHORIZATION_GATE_PATH)
r5_source_config = load_json(R5_SOURCE_CONFIG_PATH)
r5_source_result = load_json(R5_SOURCE_RESULT_GATE_PATH)
pilot_input_hashes = {
    str(path): file_sha256(path) for path in pilot_required_paths
}
r5_source_checkpoint_hash_before = pilot_input_hashes[
    str(R5_SOURCE_BEST_CHECKPOINT_PATH)
]
frozen_r5_hyperparameters = {
    'maximum_epochs': MAX_EPOCHS,
    'minimum_epochs': MINIMUM_EPOCHS,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
    'positive_patches': POSITIVE_PATCHES,
    'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
    'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
    'accumulation_steps': ACCUMULATION_STEPS,
    'base_lr': BASE_LR,
    'new_path_lr': NEW_PATH_LR,
    'gate_lr': GATE_LR,
    'weight_decay': WEIGHT_DECAY,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'ema_max_decay': EMA_MAX_DECAY,
}
pilot_upstream_checks = {
    'v22_model_preflight_passed': v22_gate.get('preflight_passed') is True,
    'v22_parameter_count_matches': int(
        v22_gate.get('total_parameters', -1)
    ) == EXPECTED_V22_PARAMETERS,
    'v22_pilot_implementation_allowed': (
        v22_gate.get('pilot_loader_implementation_authorized') is True
    ),
    'v22_pilot_was_locked': (
        v22_gate.get('pilot_training_authorized') is False
    ),
    'block1_preflight_passed': block1_gate.get('preflight_passed') is True,
    'block1_target_scope_is_fit_only': (
        block1_gate.get('target_arrays_read_splits') == ['train_fold0_fit']
    ),
    'multimodal_preflight_passed': (
        multimodal_gate.get('preflight_passed') is True
    ),
    'fivefold_preflight_passed': fivefold_gate.get('preflight_passed') is True,
    'full_fivefold_training_was_locked': (
        fivefold_gate.get('full_fivefold_training_authorized') is False
    ),
    'locked_validation_targets_sealed': (
        fivefold_gate.get('locked_validation_targets_read') is False
    ),
    'test_targets_sealed': fivefold_gate.get('test_targets_read') is False,
    'r5_replication_authorization_passed': (
        authorization_gate.get('authorization_passed') is True
    ),
    'authorization_enables_development_fivefold_training': (
        authorization_gate.get('full_fivefold_training_authorized') is True
        and authorization_gate.get(
            'development_fivefold_r5_training_authorized'
        ) is True
    ),
    'active_fold_is_in_final_protocol': (
        REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
    ),
    'fold0_restart_is_explicit_protocol_harmonization': (
        REPLICATION_FOLD != 0 or FINAL_PROTOCOL_RETRAINS_FOLD0
    ),
    'authorization_selects_r5': (
        authorization_gate.get('selected_architecture') == 'R5'
    ),
    'authorization_checkpoint_is_r5_source': (
        authorization_gate.get('selected_checkpoint_path')
        == str(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_checkpoint_hash_matches': (
        authorization_gate.get('selected_checkpoint_sha256')
        == file_sha256(R5_SOURCE_BEST_CHECKPOINT_PATH)
    ),
    'authorization_keeps_locked_validation_sealed': (
        authorization_gate.get('locked_validation_evaluation_authorized')
        is False
    ),
    'authorization_keeps_test_sealed': (
        authorization_gate.get('test_evaluation_authorized') is False
    ),
    'authorization_freezes_architecture': (
        authorization_gate.get('architecture_changes_authorized') is False
    ),
    'authorization_freezes_hyperparameters': (
        authorization_gate.get('hyperparameter_changes_authorized') is False
    ),
    'authorization_identifies_fold0_as_development_fold': (
        authorization_gate.get('fold0_artifacts_reused') is True
        and authorization_gate.get('new_training_authorized_folds')
        == [1, 2, 3, 4]
    ),
    'r5_source_pilot_complete': (
        r5_source_result.get('pilot_complete') is True
    ),
    'r5_source_parameter_count_matches': (
        int(r5_source_config.get('parameter_count', -1))
        == EXPECTED_V22_PARAMETERS
    ),
    'r5_hyperparameters_are_exactly_frozen': all(
        r5_source_config.get(name) == value
        for name, value in frozen_r5_hyperparameters.items()
    ),
    'r5_threshold_grids_are_exactly_frozen': (
        r5_source_config.get('local_thresholds')
        == LOCAL_THRESHOLDS.tolist()
        and r5_source_config.get('proposal_thresholds')
        == PROPOSAL_THRESHOLDS.tolist()
    ),
    'r5_checkpoint_score_is_exactly_frozen': (
        r5_source_config.get('checkpoint_score')
        == (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        )
    ),
    'r5_component_detection_policy_is_exactly_frozen': (
        r5_source_config.get('component_detection_policy')
        == 'final IoU >= 0.10; global proposal IoU >= 0.05'
    ),
    'target_policy_manifest_hash_matches': (
        file_sha256(TARGET_POLICY_MANIFEST_PATH)
        == EXPECTED_TARGET_POLICY_MANIFEST_SHA256
    ),
    'fold_manifest_hash_matches': (
        file_sha256(FOLD_MANIFEST_PATH) == EXPECTED_FOLD_MANIFEST_SHA256
    ),
    'multimodal_manifest_hash_matches': (
        file_sha256(MULTIMODAL_MANIFEST_PATH)
        == EXPECTED_MULTIMODAL_MANIFEST_SHA256
    ),
    'graph_schema_hash_matches': (
        file_sha256(GRAPH_SCHEMA_PATH) == EXPECTED_GRAPH_SCHEMA_SHA256
    ),
}
if not all(pilot_upstream_checks.values()):
    failed = [k for k, v in pilot_upstream_checks.items() if not bool(v)]
    raise RuntimeError(
        'TRIAL8 V2.2 PILOT UPSTREAM GATE FAILED. Do not train. Failed: '
        + ', '.join(failed)
    )

ACTIVE_GRAPH_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['graph_scaler_paths'], REPLICATION_FOLD
))
ACTIVE_CLINICAL_SCALER_PATH = Path(resolve_gate_mapping(
    multimodal_gate['clinical_scaler_paths'], REPLICATION_FOLD
))
active_scaler_checks = {
    'active_graph_scaler_exists': ACTIVE_GRAPH_SCALER_PATH.exists(),
    'active_clinical_scaler_exists': ACTIVE_CLINICAL_SCALER_PATH.exists(),
    'active_graph_scaler_hash_matches': (
        ACTIVE_GRAPH_SCALER_PATH.exists()
        and file_sha256(ACTIVE_GRAPH_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['graph_scaler_sha256'], REPLICATION_FOLD
        )
    ),
    'active_clinical_scaler_hash_matches': (
        ACTIVE_CLINICAL_SCALER_PATH.exists()
        and file_sha256(ACTIVE_CLINICAL_SCALER_PATH)
        == resolve_gate_mapping(
            multimodal_gate['clinical_scaler_sha256'], REPLICATION_FOLD
        )
    ),
}
if not all(active_scaler_checks.values()):
    failed = [name for name, passed in active_scaler_checks.items() if not passed]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler gate failed: ' + ', '.join(failed)
    )
pilot_upstream_checks.update(active_scaler_checks)
pilot_input_hashes[str(ACTIVE_GRAPH_SCALER_PATH)] = file_sha256(
    ACTIVE_GRAPH_SCALER_PATH
)
pilot_input_hashes[str(ACTIVE_CLINICAL_SCALER_PATH)] = file_sha256(
    ACTIVE_CLINICAL_SCALER_PATH
)




_pilot_uncached_file_sha256 = file_sha256
_pilot_sha256_cache = {}


def file_sha256(path, chunk_size=1024 * 1024):
    path = Path(path)
    stat = path.stat()
    key = (str(path), int(stat.st_size), int(stat.st_mtime_ns))
    if key not in _pilot_sha256_cache:
        _pilot_sha256_cache[key] = _pilot_uncached_file_sha256(
            path, chunk_size=chunk_size
        )
    return _pilot_sha256_cache[key]

with np.load(GRAPH_SCHEMA_PATH, allow_pickle=False) as data:
    pilot_schema = {name: np.asarray(data[name]) for name in data.files}

pilot_manifest_df = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
pilot_fold_df = pd.read_csv(FOLD_MANIFEST_PATH)
pilot_manifest_df['subject_id'] = pilot_manifest_df['subject_id'].astype(str)
pilot_fold_df['subject_id'] = pilot_fold_df['subject_id'].astype(str)
pilot_manifest_df = pilot_manifest_df.sort_values('subject_id').reset_index(drop=True)
pilot_fold_df = pilot_fold_df.sort_values('subject_id').reset_index(drop=True)
if pilot_manifest_df['subject_id'].tolist() != pilot_fold_df['subject_id'].tolist():
    raise RuntimeError('Multimodal and five-fold manifests disagree on subjects.')
pilot_manifest_df['development_fold'] = pilot_fold_df[
    'development_fold'
].astype(int)
development_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'train'
].copy()
active_fit_df = development_df[
    development_df['development_fold'] != REPLICATION_FOLD
].copy().sort_values('subject_id')
active_heldout_df = development_df[
    development_df['development_fold'] == REPLICATION_FOLD
].copy().sort_values('subject_id')
locked_validation_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'val'
].copy()
sealed_test_df = pilot_manifest_df[
    pilot_manifest_df['dev_split'].astype(str) == 'test'
].copy()
expected_heldout_subjects = EXPECTED_DEVELOPMENT_FOLD_COUNTS[REPLICATION_FOLD]
expected_fit_subjects = 102 - expected_heldout_subjects
if (
    len(active_fit_df), len(active_heldout_df)
) != (expected_fit_subjects, expected_heldout_subjects):
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} must contain exactly '
        f'{expected_fit_subjects} fit and {expected_heldout_subjects} '
        'held-out cases.'
    )
if (len(locked_validation_df), len(sealed_test_df)) != (23, 23):
    raise RuntimeError('Locked validation/test split counts changed.')
fit_row_by_subject = {
    str(row['subject_id']): row for row in active_fit_df.to_dict('records')
}
heldout_rows = active_heldout_df.to_dict('records')

expected_fit_subject_hash = pilot_subject_hash(active_fit_df['subject_id'])
expected_heldout_subject_hash = pilot_subject_hash(
    active_heldout_df['subject_id']
)
with np.load(ACTIVE_GRAPH_SCALER_PATH, allow_pickle=False) as data:
    graph_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read',
        ]
    }
with np.load(ACTIVE_CLINICAL_SCALER_PATH, allow_pickle=False) as data:
    clinical_scaler_metadata = {
        name: np.asarray(data[name]) for name in [
            'heldout_fold', 'fit_subject_count', 'heldout_subject_count',
            'fit_subject_hash', 'heldout_subject_hash',
            'target_arrays_read', 'outcome_workbooks_read',
        ]
    }
fold_scaler_partition_checks = {
    'graph_scaler_heldout_fold_matches': int(
        graph_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'clinical_scaler_heldout_fold_matches': int(
        clinical_scaler_metadata['heldout_fold'].reshape(-1)[0]
    ) == REPLICATION_FOLD,
    'graph_scaler_partition_counts_match': (
        int(graph_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
    ),
    'clinical_scaler_partition_counts_match': (
        int(clinical_scaler_metadata['fit_subject_count'].reshape(-1)[0])
        == len(active_fit_df)
        and int(
            clinical_scaler_metadata['heldout_subject_count'].reshape(-1)[0]
        ) == len(active_heldout_df)
    ),
    'graph_scaler_subject_hashes_match': (
        scalar_string(graph_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(graph_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'clinical_scaler_subject_hashes_match': (
        scalar_string(clinical_scaler_metadata['fit_subject_hash'])
        == expected_fit_subject_hash
        and scalar_string(clinical_scaler_metadata['heldout_subject_hash'])
        == expected_heldout_subject_hash
    ),
    'graph_scaler_did_not_read_targets': not as_bool(
        graph_scaler_metadata['target_arrays_read'].reshape(-1)[0]
    ),
    'clinical_scaler_did_not_read_targets_or_outcomes': (
        not as_bool(
            clinical_scaler_metadata['target_arrays_read'].reshape(-1)[0]
        )
        and not as_bool(
            clinical_scaler_metadata['outcome_workbooks_read'].reshape(-1)[0]
        )
    ),
}
if not all(fold_scaler_partition_checks.values()):
    failed = [
        name for name, passed in fold_scaler_partition_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} scaler-partition audit failed: '
        + ', '.join(failed)
    )
pilot_upstream_checks.update(fold_scaler_partition_checks)






def pilot_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return {
        'shape': tuple(int(v) for v in mask.shape),
        'bits': np.packbits(mask.reshape(-1), bitorder='little'),
    }


def pilot_unpack_mask(record):
    count = int(np.prod(record['shape']))
    values = np.unpackbits(record['bits'], bitorder='little')[:count]
    return values.reshape(record['shape']).astype(bool)


pilot_mask_records = {}
pilot_target_rows = []
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        required = {'target_binary', 'target_soft', 'crop_mask', 'target_spacing'}
        missing = sorted(required - set(data.files))
        if missing:
            raise RuntimeError(f'{subject_id}: target arrays missing {missing}.')
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        crop_mask = np.asarray(data['crop_mask'], dtype=np.uint8) > 0
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if binary.shape != soft.shape or binary.shape != crop_mask.shape:
        raise RuntimeError(f'{subject_id}: target/crop shapes disagree.')
    if not np.isfinite(soft).all() or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: target metadata is non-finite.')
    support = binary | (soft > 0)
    pilot_mask_records[subject_id] = {
        'support': pilot_pack_mask(support),
        'crop_mask': pilot_pack_mask(crop_mask),
        'shape': tuple(int(v) for v in support.shape),
    }
    pilot_target_rows.append({
        'subject_id': subject_id,
        'development_partition': f'fold{REPLICATION_FOLD}_fit',
        'hard_voxels': int(binary.sum()),
        'soft_support_voxels': int((soft > 0).sum()),
        'support_voxels': int(support.sum()),
        'target_arrays_read': True,
        'locked_validation_target_read': False,
        'test_target_read': False,
    })
atomic_write_csv(pd.DataFrame(pilot_target_rows), PILOT_TARGET_AUDIT_PATH)


def pilot_component_tier(volume_ml, voxels, voxel_volume_ml):
    if voxels <= 1 or volume_ml <= 1.5 * voxel_volume_ml:
        return 'subvoxel_or_single_voxel'
    if volume_ml <= 0.10:
        return 'micro'
    if volume_ml <= 1.0:
        return 'tiny'
    if volume_ml <= 5.0:
        return 'small'
    if volume_ml <= 25.0:
        return 'medium'
    return 'large'






component_rows = []
connectivity_26n = np.ones((3, 3, 3), dtype=bool)
for row in active_fit_df.to_dict('records'):
    subject_id = str(row['subject_id'])
    with np.load(
        Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False
    ) as data:
        binary = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(
            data['target_spacing'], dtype=np.float64
        ).reshape(-1)
    supervision_mode = str(row['trial8_supervision_mode'])
    if supervision_mode == 'hard_and_soft':
        component_support = binary
    elif supervision_mode == 'soft_only_subvoxel':
        component_support = soft > 0
    elif supervision_mode == 'true_empty':
        component_support = np.zeros_like(binary, dtype=bool)
    else:
        raise RuntimeError(
            f'{subject_id}: unknown supervision mode '
            f'{supervision_mode!r}.'
        )
    if spacing.shape != (3,) or not np.isfinite(spacing).all():
        raise RuntimeError(f'{subject_id}: invalid target spacing.')
    if bool(component_support.any()) != as_bool(
        row['trial8_sampling_positive']
    ):
        raise RuntimeError(
            f'{subject_id}: component support disagrees with the frozen '
            'sampling-positive policy.'
        )
    labels, component_count = ndimage.label(
        component_support, structure=connectivity_26n
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    for component_id in range(1, int(component_count) + 1):
        coordinates = np.argwhere(labels == component_id)
        if len(coordinates) == 0:
            raise RuntimeError(f'{subject_id}: empty atomic component.')
        centroid = coordinates.mean(axis=0)
        voxels = int(len(coordinates))
        volume_ml = float(voxels * voxel_volume_ml)
        component_rows.append({
            'subject_id': subject_id,
            'development_fold': int(row['development_fold']),
            'supervision_mode': supervision_mode,
            'atomic_component_id': int(component_id),
            'component_tier': pilot_component_tier(
                volume_ml, voxels, voxel_volume_ml
            ),
            'component_voxels': voxels,
            'component_volume_ml': volume_ml,
            'centroid_x_vox': float(centroid[0]),
            'centroid_y_vox': float(centroid[1]),
            'centroid_z_vox': float(centroid[2]),
            'target_arrays_read': True,
        })

component_df = pd.DataFrame(component_rows)
if component_df.empty:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} fitting partition has no target components.'
    )
component_df['subject_id'] = component_df['subject_id'].astype(str)
if set(component_df['subject_id']) - set(fit_row_by_subject):
    raise RuntimeError('Component audit contains non-fitting subjects.')
atomic_write_csv(component_df, PILOT_COMPONENT_AUDIT_PATH)
component_records = {
    subject_id: group.sort_values('atomic_component_id').to_dict('records')
    for subject_id, group in component_df.groupby('subject_id')
}
positive_subjects = sorted(
    subject_id for subject_id in fit_row_by_subject
    if len(component_records.get(subject_id, [])) > 0
)
expected_positive_subjects = sorted(
    str(row['subject_id'])
    for row in active_fit_df.to_dict('records')
    if as_bool(row['trial8_sampling_positive'])
)
if positive_subjects != expected_positive_subjects:
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} reconstructed positive-subject set does '
        'not match the frozen sampling policy.'
    )

fit_subject_ids = set(active_fit_df['subject_id'].astype(str))
heldout_subject_ids = set(active_heldout_df['subject_id'].astype(str))
locked_validation_subject_ids = set(
    locked_validation_df['subject_id'].astype(str)
)
sealed_test_subject_ids = set(sealed_test_df['subject_id'].astype(str))
audited_target_subject_ids = {
    str(row['subject_id']) for row in pilot_target_rows
}
pilot_target_access_checks = {
    'fit_and_heldout_subjects_are_disjoint': not bool(
        fit_subject_ids & heldout_subject_ids
    ),
    'fit_and_heldout_cover_all_development_subjects': (
        fit_subject_ids | heldout_subject_ids
        == set(development_df['subject_id'].astype(str))
    ),
    'pretraining_target_audit_is_exactly_active_fit': (
        audited_target_subject_ids == fit_subject_ids
    ),
    'component_audit_is_active_fit_only': (
        set(component_df['subject_id'].astype(str)) <= fit_subject_ids
    ),
    'heldout_targets_not_read_during_pretraining_audit': not bool(
        audited_target_subject_ids & heldout_subject_ids
    ),
    'locked_validation_targets_not_read': not bool(
        audited_target_subject_ids & locked_validation_subject_ids
    ),
    'test_targets_not_read': not bool(
        audited_target_subject_ids & sealed_test_subject_ids
    ),
}
if not all(pilot_target_access_checks.values()):
    failed = [
        name for name, passed in pilot_target_access_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f'Fold-{REPLICATION_FOLD} target-access audit failed: '
        + ', '.join(failed)
    )


def pilot_origin_from_center(center_native, native_shape):
    pad_start = (
        (np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(native_shape)) // 2
    )
    center_canvas = pad_start + np.asarray(center_native, dtype=np.int64)
    origin = center_canvas - np.asarray(LOCAL_PATCH_SIZE) // 2
    maximum = np.asarray(GLOBAL_CANVAS_SHAPE) - np.asarray(LOCAL_PATCH_SIZE)
    return tuple(int(v) for v in np.clip(origin, 0, maximum))


def pilot_native_patch_slices(shape, center):
    slices = []
    for size, coordinate, patch in zip(shape, center, LOCAL_PATCH_SIZE):
        start = int(round(float(coordinate))) - patch // 2
        start = min(max(start, 0), max(int(size) - patch, 0))
        slices.append(slice(start, min(start + patch, int(size))))
    return tuple(slices)


def pilot_target_free_center(subject_id, rng, topology=False):
    support = pilot_unpack_mask(pilot_mask_records[subject_id]['support'])
    crop_mask = pilot_unpack_mask(pilot_mask_records[subject_id]['crop_mask'])
    integral = np.pad(
        support.astype(np.int32).cumsum(0).cumsum(1).cumsum(2),
        ((1, 0), (1, 0), (1, 0)), mode='constant',
    )

    def support_in_box(slices):
        x0, x1 = slices[0].start, slices[0].stop
        y0, y1 = slices[1].start, slices[1].stop
        z0, z1 = slices[2].start, slices[2].stop
        return int(
            integral[x1, y1, z1]
            - integral[x0, y1, z1]
            - integral[x1, y0, z1]
            - integral[x1, y1, z0]
            + integral[x0, y0, z1]
            + integral[x0, y1, z0]
            + integral[x1, y0, z0]
            - integral[x0, y0, z0]
        )
    if topology:
        row = fit_row_by_subject[subject_id]
        if as_bool(row['trial8_atlas_enabled']):
            atlas = np.asanyarray(
                nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
            )
            if atlas.ndim == 4:
                atlas = atlas[..., 0]
            candidate = crop_mask & (np.rint(atlas).astype(np.int16) > 0)
        else:
            candidate = crop_mask
    else:
        candidate = crop_mask
    coordinates = np.argwhere(candidate)
    if len(coordinates) == 0:
        raise RuntimeError(f'{subject_id}: no negative-center candidates.')
    search_count = min(len(coordinates), 4096)
    order = rng.choice(len(coordinates), size=search_count, replace=False)
    
    
    
    
    
    for index in order:
        center = coordinates[int(index)]
        slices = pilot_native_patch_slices(support.shape, center)
        if support_in_box(slices) == 0:
            return tuple(int(v) for v in center)
    
    
    
    starts_by_axis = []
    for size, patch in zip(support.shape, LOCAL_PATCH_SIZE):
        maximum = max(int(size) - int(patch), 0)
        starts = list(range(0, maximum + 1, 4))
        if not starts or starts[-1] != maximum:
            starts.append(maximum)
        starts_by_axis.append(starts)
    for x0 in starts_by_axis[0]:
        for y0 in starts_by_axis[1]:
            for z0 in starts_by_axis[2]:
                slices = (
                    slice(x0, min(x0 + LOCAL_PATCH_SIZE[0], support.shape[0])),
                    slice(y0, min(y0 + LOCAL_PATCH_SIZE[1], support.shape[1])),
                    slice(z0, min(z0 + LOCAL_PATCH_SIZE[2], support.shape[2])),
                )
                if support_in_box(slices) != 0:
                    continue
                center = tuple(
                    min(
                        start + LOCAL_PATCH_SIZE[axis] // 2,
                        support.shape[axis] - 1,
                    )
                    for axis, start in enumerate((x0, y0, z0))
                )
                if bool(candidate[center]):
                    return tuple(int(v) for v in center)
    return None


def pilot_choose_component(subject_id, epoch, occurrence, rng):
    records = component_records[subject_id]
    if occurrence == 0:
        weights = np.asarray([
            max(float(item['component_volume_ml']), 1e-6) ** -0.35
            for item in records
        ], dtype=np.float64)
    else:
        tier_counts = defaultdict(int)
        for item in component_df.to_dict('records'):
            tier_counts[str(item['component_tier'])] += 1
        weights = np.asarray([
            1.0 / max(tier_counts[str(item['component_tier'])], 1)
            for item in records
        ], dtype=np.float64)
    weights /= weights.sum()
    return records[int(rng.choice(len(records), p=weights))]


def pilot_build_epoch_schedule(epoch, return_skip_audit=False):
    rng = np.random.default_rng(PILOT_SEED + int(epoch) * 1_000_003)
    entries = []
    occurrence = defaultdict(int)
    base_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order = base_order.copy()
    extra_order = list(np.asarray(positive_subjects, dtype=object)[
        rng.permutation(len(positive_subjects))
    ])
    positive_order.extend(extra_order[:POSITIVE_PATCHES - len(positive_order)])
    for subject_id in positive_order:
        component = pilot_choose_component(
            str(subject_id), epoch, occurrence[str(subject_id)], rng
        )
        center = (
            float(component['centroid_x_vox']),
            float(component['centroid_y_vox']),
            float(component['centroid_z_vox']),
        )
        jitter = np.asarray([
            rng.integers(-12, 13), rng.integers(-12, 13), rng.integers(-5, 6)
        ])
        center = tuple(
            int(np.clip(round(center[i]) + jitter[i], 0,
                        pilot_mask_records[str(subject_id)]['shape'][i] - 1))
            for i in range(3)
        )
        entries.append({
            'sample_type': 'positive', 'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': int(component['atomic_component_id']),
            'component_tier': str(component['component_tier']),
            'component_volume_ml': float(component['component_volume_ml']),
        })
        occurrence[str(subject_id)] += 1
    skip_rows = []
    topology_pool = sorted(
        subject_id for subject_id, row in fit_row_by_subject.items()
        if as_bool(row['trial8_atlas_enabled'])
    )
    topology_pool = list(np.asarray(topology_pool, dtype=object)[
        rng.permutation(len(topology_pool))
    ])
    topology_added = 0
    for subject_id in topology_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=True
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'topology_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'topology_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        topology_added += 1
        if topology_added == TOPOLOGY_NEGATIVE_PATCHES:
            break
    if topology_added != TOPOLOGY_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete atlas-enabled fold-{REPLICATION_FOLD} fit pool '
            'supplied only '
            f'{topology_added}/{TOPOLOGY_NEGATIVE_PATCHES} target-free '
            'topology-negative patches.'
        )
    brain_pool = sorted(fit_row_by_subject)
    brain_pool = list(np.asarray(brain_pool, dtype=object)[
        rng.permutation(len(brain_pool))
    ])
    brain_added = 0
    for subject_id in brain_pool:
        center = pilot_target_free_center(
            str(subject_id), rng, topology=False
        )
        if center is None:
            skip_rows.append({
                'epoch': int(epoch), 'negative_role': 'brain_negative',
                'subject_id': str(subject_id),
                'reason': 'no_target_free_128x128x64_patch',
            })
            continue
        entries.append({
            'sample_type': 'brain_negative',
            'subject_id': str(subject_id),
            'center_native': center,
            'atomic_component_id': 0, 'component_tier': 'none',
            'component_volume_ml': 0.0,
        })
        brain_added += 1
        if brain_added == BRAIN_NEGATIVE_PATCHES:
            break
    if brain_added != BRAIN_NEGATIVE_PATCHES:
        raise RuntimeError(
            f'The complete fold-{REPLICATION_FOLD} fit pool supplied only '
            f'{brain_added}/{BRAIN_NEGATIVE_PATCHES} target-free '
            'brain-negative patches.'
        )
    rng.shuffle(entries)
    if len(entries) != 168:
        raise RuntimeError(f'Pilot schedule has {len(entries)} not 168 samples.')
    if return_skip_audit:
        return entries, skip_rows
    return entries


schedule_a, schedule_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
schedule_b, repeated_skip_audit = pilot_build_epoch_schedule(
    1, return_skip_audit=True
)
if schedule_a != schedule_b:
    raise RuntimeError('Pilot epoch schedule is not deterministic.')
if schedule_skip_audit != repeated_skip_audit:
    raise RuntimeError('Negative-candidate skip audit is not deterministic.')
atomic_write_csv(pd.DataFrame(schedule_a), PILOT_SCHEDULE_AUDIT_PATH)
atomic_write_csv(
    pd.DataFrame(
        schedule_skip_audit,
        columns=['epoch', 'negative_role', 'subject_id', 'reason'],
    ),
    PILOT_NEGATIVE_SKIP_AUDIT_PATH,
)






MODEL_INPUT_KEYS_V22 = [
    'image', 'region_map', 'territory_features', 'territory_feature_valid',
    'territory_valid', 'territory_reliability', 'vessel_features',
    'vessel_feature_valid', 'vessel_valid', 'case_features', 'atlas_enabled',
    'vascular_image', 'confidence_features', 'occlusion_values',
    'occlusion_valid', 'clinical_input', 'vascular_available',
    'occlusion_available', 'clinical_available', 'global_image',
    'global_region_map', 'local_origin_in_canvas',
]


def pilot_extract_targets(row, local_origin):
    subject_id = str(row['subject_id'])
    with np.load(Path(str(row['vascufate_v2_npz_path'])), allow_pickle=False) as data:
        binary = np.asarray(data['target_binary'], dtype=np.float32)
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    support = np.maximum(binary, soft).astype(np.float32)
    binary_canvas, start_a = symmetric_pad(
        binary, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    soft_canvas, start_b = symmetric_pad(
        soft, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if start_a != start_b:
        raise RuntimeError(f'{subject_id}: target padding starts disagree.')
    local_binary = crop_at(
        binary_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    local_soft = crop_at(
        soft_canvas, local_origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    )
    support_canvas = np.maximum(binary_canvas, soft_canvas)
    global_target = F.max_pool3d(
        torch.from_numpy(support_canvas[None, None]),
        kernel_size=GLOBAL_DOWNSAMPLE, stride=GLOBAL_DOWNSAMPLE,
    )[0].numpy().astype(np.float32)
    volume_ml = float(
        soft.sum(dtype=np.float64) * np.prod(spacing) / 1000.0
    )
    return {
        'target_binary': local_binary[None].astype(np.float32),
        'target_soft': local_soft[None].astype(np.float32),
        'global_target': global_target,
        'presence_target': np.asarray(float(bool((support > 0).any())), np.float32),
        'log_volume_target': np.asarray(np.log1p(volume_ml), np.float32),
        'spacing': spacing.astype(np.float32),
        'native_support_voxels': int((support > 0).sum()),
    }


def pilot_apply_intensity_augmentation(sample, rng):
    image = sample['image'].copy()
    global_image = sample['global_image'].copy()
    intensity_channels = 13
    gains = rng.uniform(0.94, 1.06, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    offsets = rng.uniform(-0.015, 0.015, size=(intensity_channels, 1, 1, 1)).astype(
        np.float32
    )
    image[:intensity_channels] = image[:intensity_channels] * gains + offsets
    global_image[:intensity_channels] = (
        global_image[:intensity_channels] * gains + offsets
    )
    if rng.random() < 0.35:
        noise_std = float(rng.uniform(0.0, 0.010))
        image[:intensity_channels] += rng.normal(
            0.0, noise_std, image[:intensity_channels].shape
        ).astype(np.float32)
        global_image[:intensity_channels] += rng.normal(
            0.0, noise_std, global_image[:intensity_channels].shape
        ).astype(np.float32)
    image[:9] = np.clip(image[:9], 0.0, 1.0)
    global_image[:9] = np.clip(global_image[:9], 0.0, 1.0)
    image[9:13] = np.clip(image[9:13], -1.0, 1.0)
    global_image[9:13] = np.clip(global_image[9:13], -1.0, 1.0)
    sample['image'] = image.astype(np.float32)
    sample['global_image'] = global_image.astype(np.float32)
    cta_index = EXPECTED_INPUT_CHANNELS.index('cta')
    tmax_index = EXPECTED_INPUT_CHANNELS.index('tmax')
    sample['vascular_image'][0] = sample['image'][cta_index]
    sample['vascular_image'][1] = sample['image'][tmax_index]
    return sample


class Trial8V22ReplicationDataset(torch.utils.data.Dataset):
    def __init__(self):
        self.epoch = 1
        self.schedule = []
        self.set_epoch(1)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)
        self.schedule = pilot_build_epoch_schedule(self.epoch)

    def __len__(self):
        return len(self.schedule)

    def __getitem__(self, index):
        entry = self.schedule[index]
        subject_id = str(entry['subject_id'])
        row = fit_row_by_subject[subject_id]
        native_shape = pilot_mask_records[subject_id]['shape']
        local_origin = pilot_origin_from_center(
            entry['center_native'], native_shape
        )
        sample, metadata = load_v22_subject(
            row, pilot_schema, multimodal_gate, local_origin=local_origin
        )
        targets = pilot_extract_targets(row, local_origin)
        if entry['sample_type'] == 'positive' and not bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: positive patch lost soft support.')
        if entry['sample_type'] != 'positive' and bool(
            (targets['target_soft'] > 0).any()
        ):
            raise RuntimeError(f'{subject_id}: negative patch contains target.')
        rng = np.random.default_rng(
            PILOT_SEED + self.epoch * 10_000_019 + index * 104_729
        )
        sample = pilot_apply_intensity_augmentation(sample, rng)
        supervision_mode = str(row['trial8_supervision_mode'])
        hard_weight = 0.0 if supervision_mode == 'soft_only_subvoxel' else 1.0
        sample.update({
            'target_binary': targets['target_binary'],
            'target_soft': targets['target_soft'],
            'global_target': targets['global_target'],
            'presence_target': targets['presence_target'],
            'log_volume_target': targets['log_volume_target'],
            'hard_weight': np.asarray(hard_weight, dtype=np.float32),
            'subject_id': subject_id,
            'sample_type': str(entry['sample_type']),
            'component_tier': str(entry['component_tier']),
            'atomic_component_id': np.asarray(
                entry['atomic_component_id'], dtype=np.int64
            ),
        })
        return sample


def pilot_to_device(cpu_batch, device):
    output = {}
    for key, value in cpu_batch.items():
        output[key] = value.to(device, non_blocking=True) if torch.is_tensor(value) else value
    return output


def pilot_soft_dice_loss(logits, target, epsilon=1.0):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    intersection = (probability * target).sum(dim=dimensions)
    denominator = probability.sum(dim=dimensions) + target.sum(dim=dimensions)
    return 1.0 - (2.0 * intersection + epsilon) / (denominator + epsilon)


def pilot_focal_bce_loss(logits, target, gamma=2.0):
    logits = logits.float()
    target = target.float()
    probability = torch.sigmoid(logits)
    raw = F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    modulation = torch.abs(target - probability).pow(gamma)
    alpha = target * 0.75 + (1.0 - target) * 0.25
    dimensions = tuple(range(1, logits.ndim))
    return (raw * modulation * alpha).mean(dim=dimensions)


def pilot_focal_tversky_loss(
    logits, target, alpha=0.30, beta=0.70, gamma=0.75
):
    probability = torch.sigmoid(logits.float())
    target = target.float()
    dimensions = tuple(range(1, probability.ndim))
    true_positive = (probability * target).sum(dim=dimensions)
    false_positive = (probability * (1.0 - target)).sum(dim=dimensions)
    false_negative = ((1.0 - probability) * target).sum(dim=dimensions)
    tversky = (true_positive + 1.0) / (
        true_positive + alpha * false_positive + beta * false_negative + 1.0
    )
    return (1.0 - tversky).clamp_min(1e-6).pow(gamma)


def pilot_segmentation_loss(logits, soft_target, binary_target, hard_weight):
    soft_dice = pilot_soft_dice_loss(logits, soft_target)
    soft_bce = pilot_focal_bce_loss(logits, soft_target)
    hard_tversky = pilot_focal_tversky_loss(logits, binary_target)
    total = (
        0.50 * soft_dice + 0.30 * soft_bce
        + 0.20 * hard_tversky * hard_weight.reshape(-1)
    )
    return total.mean(), {
        'soft_dice_loss': soft_dice.mean(),
        'soft_focal_bce_loss': soft_bce.mean(),
        'hard_tversky_loss': hard_tversky.mean(),
    }


def pilot_territory_target(region_map, target):
    result = target.new_zeros((target.shape[0], NUM_TERRITORY_NODES))
    for batch_index in range(target.shape[0]):
        lesion = target[batch_index, 0] > 0
        if bool(lesion.any()):
            labels = torch.unique(region_map[batch_index][lesion])
            labels = labels[(labels >= 1) & (labels <= NUM_TERRITORY_NODES)]
            if len(labels):
                result[batch_index, labels.long() - 1] = 1.0
    return result


def pilot_masked_node_loss(logits, targets, valid):
    if not bool(valid.any()):
        return logits.sum() * 0.0
    raw = F.binary_cross_entropy_with_logits(
        logits.float(), targets.float(), reduction='none',
        pos_weight=torch.full(
            (NUM_TERRITORY_NODES,), 4.0,
            device=logits.device, dtype=torch.float32,
        ),
    )
    return raw[valid].mean()


def pilot_dropout_flags(rng):
    atlas = bool(rng.random() < 0.15)
    return {
        'force_atlas_dropout': atlas,
        'force_vascular_dropout': bool(rng.random() < 0.10),
        'force_occlusion_dropout': bool(rng.random() < 0.15),
        'force_clinical_dropout': bool(rng.random() < 0.20),
        'force_global_dropout': bool(rng.random() < 0.10),
        'force_arterial_dropout': atlas or bool(rng.random() < 0.10),
    }


def pilot_compute_training_loss(model, cpu_batch, device, epoch, batch_index):
    batch = pilot_to_device(cpu_batch, device)
    dropout_rng = np.random.default_rng(
        PILOT_SEED + int(epoch) * 1_000_003 + int(batch_index) * 97_409
    )
    flags = pilot_dropout_flags(dropout_rng)
    output = model(
        **{key: batch[key] for key in MODEL_INPUT_KEYS_V22}, **flags
    )
    main_loss, main_parts = pilot_segmentation_loss(
        output['logits'], batch['target_soft'],
        batch['target_binary'], batch['hard_weight'],
    )
    half_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_half_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    half_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_half_logits'].shape[2:],
        mode='nearest',
    )
    quarter_soft = F.interpolate(
        batch['target_soft'], output['auxiliary_quarter_logits'].shape[2:],
        mode='trilinear', align_corners=False,
    )
    quarter_binary = F.interpolate(
        batch['target_binary'], output['auxiliary_quarter_logits'].shape[2:],
        mode='nearest',
    )
    half_loss, _ = pilot_segmentation_loss(
        output['auxiliary_half_logits'], half_soft, half_binary,
        batch['hard_weight'],
    )
    quarter_loss, _ = pilot_segmentation_loss(
        output['auxiliary_quarter_logits'], quarter_soft, quarter_binary,
        batch['hard_weight'],
    )
    local_targets = pilot_territory_target(
        batch['region_map'], batch['target_binary'] > 0.5
    )
    local_valid = (
        batch['territory_valid'].bool() & batch['atlas_enabled'][:, None].bool()
    )
    node_loss = pilot_masked_node_loss(
        output['territory_node_logits'], local_targets, local_valid
    )
    global_target = batch['global_target'].float()
    proposal_dice = pilot_soft_dice_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_bce = pilot_focal_bce_loss(
        output['global_proposal_logits'], global_target
    ).mean()
    proposal_loss = 0.65 * proposal_dice + 0.35 * proposal_bce
    presence_loss = F.binary_cross_entropy_with_logits(
        output['lesion_presence_logit'].float(),
        batch['presence_target'].float(),
    )
    volume_mask = batch['presence_target'].float() > 0.5
    if bool(volume_mask.any()):
        volume_loss = F.smooth_l1_loss(
            output['log_volume_prediction'].float()[volume_mask],
            batch['log_volume_target'].float()[volume_mask], beta=0.5,
        )
    else:
        volume_loss = output['log_volume_prediction'].sum() * 0.0
    global_territory_target = pilot_territory_target(
        batch['global_region_map'], global_target > 0
    )
    arterial_valid = output['arterial_territory_valid'].bool()
    arterial_loss = pilot_masked_node_loss(
        output['arterial_territory_logits'], global_territory_target,
        arterial_valid,
    )
    total = (
        main_loss + 0.15 * half_loss + 0.10 * quarter_loss
        + 0.02 * node_loss + 0.35 * proposal_loss
        + 0.05 * presence_loss + 0.05 * volume_loss
        + 0.03 * arterial_loss
    )
    parts = {
        'total_loss': total, 'main_loss': main_loss,
        **main_parts, 'half_loss': half_loss, 'quarter_loss': quarter_loss,
        'node_loss': node_loss, 'proposal_loss': proposal_loss,
        'proposal_dice_loss': proposal_dice,
        'proposal_focal_bce_loss': proposal_bce,
        'presence_loss': presence_loss, 'volume_loss': volume_loss,
        'arterial_loss': arterial_loss,
        'atlas_dropped': torch.as_tensor(float(flags['force_atlas_dropout']), device=device),
        'global_dropped': torch.as_tensor(float(flags['force_global_dropout']), device=device),
        'clinical_dropped': torch.as_tensor(float(flags['force_clinical_dropout']), device=device),
    }
    return total, parts






def pilot_gaussian_weight():
    axes = [
        np.linspace(-1.0, 1.0, size, dtype=np.float32)
        for size in LOCAL_PATCH_SIZE
    ]
    grids = np.meshgrid(*axes, indexing='ij')
    radius = sum(grid ** 2 for grid in grids)
    return np.maximum(
        np.exp(-radius / (2.0 * 0.5 ** 2)).astype(np.float32), 0.05
    )


PILOT_WINDOW_WEIGHT = pilot_gaussian_weight()


def pilot_sliding_starts(size, patch, overlap=VALIDATION_OVERLAP):
    if size <= patch:
        return [0]
    stride = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, stride))
    if starts[-1] != size - patch:
        starts.append(size - patch)
    return starts


def pilot_stack_sample(sample, device):
    return {
        key: torch.from_numpy(np.asarray(value)[None]).to(device)
        for key, value in sample.items()
    }


def pilot_global_cache(
    model, batch, force_atlas_dropout=False,
    force_clinical_dropout=False, force_occlusion_dropout=False,
    force_arterial_dropout=False,
):
    atlas = batch['atlas_enabled'].bool()
    effective_atlas = atlas & ~torch.full_like(
        atlas, bool(force_atlas_dropout)
    )
    effective_arterial = effective_atlas & ~torch.full_like(
        effective_atlas, bool(force_arterial_dropout)
    )
    effective_clinical = batch['clinical_available'].bool() & ~torch.full_like(
        batch['clinical_available'].bool(), bool(force_clinical_dropout)
    )
    effective_occlusion = batch['occlusion_available'].bool() & ~torch.full_like(
        batch['occlusion_available'].bool(), bool(force_occlusion_dropout)
    )
    arterial = model.arterial_projector(
        batch['vessel_features'], batch['vessel_feature_valid'],
        batch['vessel_valid'].bool() & effective_arterial[:, None],
        batch['territory_valid'], batch['territory_reliability'],
        effective_arterial,
    )
    arterial_map, arterial_map_valid = territory_values_to_spatial(
        arterial['territory_risk_logits'],
        arterial['territory_projection_valid'], batch['global_region_map'],
    )
    arterial_map = arterial_map * arterial_map_valid.to(arterial_map.dtype)
    global_input = torch.cat([
        batch['global_image'], arterial_map[:, None],
        arterial_map_valid[:, None].to(batch['global_image'].dtype),
    ], dim=1)
    skips, bottleneck, image_context = model.global_stream.encode(global_input)
    routed = model.global_router(
        image_context,
        arterial['arterial_global_context']
        * effective_arterial[:, None].to(image_context.dtype),
        batch['clinical_input']
        * effective_clinical[:, None].to(image_context.dtype),
        batch['occlusion_values'], batch['occlusion_valid'],
        batch['confidence_features'], effective_atlas,
        effective_occlusion, effective_clinical,
    )
    proposal_logits, global_feature = model.global_stream.decode(
        skips, bottleneck, routed['global_context']
    )
    return {
        'proposal_logits': proposal_logits,
        'global_feature': global_feature,
        'arterial_map': arterial_map,
        'arterial': arterial,
        'routed': routed,
        'effective_atlas': effective_atlas,
        'effective_arterial': effective_arterial,
        'effective_clinical': effective_clinical,
        'effective_occlusion': effective_occlusion,
    }


def pilot_local_from_cache(
    model, batch, cache, force_global_dropout=False,
    force_atlas_dropout=False, force_vascular_dropout=False,
    force_occlusion_dropout=False, force_clinical_dropout=False,
):
    proposal_local = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    feature_local = sample_global_to_local(
        cache['global_feature'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    global_enabled = ~torch.full_like(
        batch['atlas_enabled'].bool(), bool(force_global_dropout)
    )
    local_confidence = torch.cat([
        batch['confidence_features'],
        torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
    ], dim=-1)
    conditioned_image, proposal_gate = model.local_fusion(
        batch['image'], proposal_local, feature_local, arterial_local,
        local_confidence, global_enabled,
    )
    local = model.local_model(
        image=conditioned_image,
        region_map=batch['region_map'],
        territory_features=batch['territory_features'],
        territory_feature_valid=batch['territory_feature_valid'],
        territory_valid=batch['territory_valid'],
        territory_reliability=batch['territory_reliability'],
        vessel_features=batch['vessel_features'],
        vessel_feature_valid=batch['vessel_feature_valid'],
        vessel_valid=batch['vessel_valid'],
        case_features=batch['case_features'],
        atlas_enabled=batch['atlas_enabled'],
        vascular_image=batch['vascular_image'],
        confidence_features=batch['confidence_features'],
        occlusion_values=batch['occlusion_values'],
        occlusion_valid=batch['occlusion_valid'],
        clinical_input=batch['clinical_input'],
        vascular_available=batch['vascular_available'],
        occlusion_available=batch['occlusion_available'],
        clinical_available=batch['clinical_available'],
        force_atlas_dropout=force_atlas_dropout,
        force_vascular_dropout=force_vascular_dropout,
        force_occlusion_dropout=force_occlusion_dropout,
        force_clinical_dropout=force_clinical_dropout,
    )
    local.update({
        'global_proposal_logits': cache['proposal_logits'],
        'lesion_presence_logit': cache['routed']['lesion_presence_logit'],
        'log_volume_prediction': cache['routed']['log_volume_prediction'],
        'proposal_fusion_gate': proposal_gate,
    })
    return local


def pilot_load_validation_case(row):
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
        target = np.asarray(data['target_binary'], dtype=np.uint8) > 0
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(image.shape[1:], dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: validation padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id, 'row': row, 'image_canvas': image_canvas,
        'region_canvas': region_canvas, 'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas, 'lvo_present': lvo_present,
        'target': target, 'soft': soft, 'spacing': spacing,
        'native_shape': tuple(int(v) for v in target.shape),
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


def pilot_window_sample(case, origin):
    sample = {
        key: np.asarray(value).copy()
        for key, value in case['base_sample'].items()
    }
    image_patch = crop_at(
        case['image_canvas'], origin, LOCAL_PATCH_SIZE, (1, 2, 3)
    ).astype(np.float32)
    region_patch = crop_at(
        case['region_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.int64)
    cow_patch = crop_at(
        case['cow_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    lvo_patch = crop_at(
        case['lvo_canvas'], origin, LOCAL_PATCH_SIZE, (0, 1, 2)
    ).astype(np.float32)
    sample['image'] = image_patch
    sample['region_map'] = region_patch
    sample['vascular_image'] = np.stack([
        image_patch[EXPECTED_INPUT_CHANNELS.index('cta')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('tmax')],
        image_patch[EXPECTED_INPUT_CHANNELS.index('perfusion_valid')],
        cow_patch, lvo_patch,
    ]).astype(np.float32)
    sample['confidence_features'] = sample['confidence_features'].copy()
    sample['confidence_features'][0] = float(sample['vascular_image'][2].mean())
    sample['occlusion_values'] = sample['occlusion_values'].copy()
    sample['occlusion_values'][1] = float(lvo_patch.max())
    sample['local_origin_in_canvas'] = np.asarray(origin, dtype=np.int64)
    return sample


@torch.no_grad()
def pilot_infer_case(model, row, device, ablation='none'):
    case = pilot_load_validation_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    force_atlas = ablation == 'atlas'
    force_arterial = ablation in {'arterial', 'atlas'}
    force_clinical = ablation == 'clinical'
    force_occlusion = ablation == 'occlusion'
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(
            model, base_batch, force_atlas_dropout=force_atlas,
            force_clinical_dropout=force_clinical,
            force_occlusion_dropout=force_occlusion,
            force_arterial_dropout=force_arterial,
        )
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][i] + native_start[i])
                    for i in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(
                        model, batch, cache,
                        force_global_dropout=(ablation == 'global'),
                        force_atlas_dropout=force_atlas,
                        force_vascular_dropout=(ablation == 'vascular'),
                        force_occlusion_dropout=force_occlusion,
                        force_clinical_dropout=force_clinical,
                    )
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(native_start[i], min(
                        native_start[i] + LOCAL_PATCH_SIZE[i], native_shape[i]
                    )) for i in range(3)
                )
                copied_shape = tuple(s.stop - s.start for s in destination)
                source = tuple(slice(0, copied_shape[i]) for i in range(3))
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(f"{case['subject_id']}: incomplete inference coverage.")
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(case['pad_start'][i], case['pad_start'][i] + native_shape[i])
        for i in range(3)
    )
    proposal = proposal_canvas[native_slices]
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float16),
        'proposal_probability': proposal.astype(np.float16),
        'target': case['target'], 'soft': case['soft'],
        'spacing': case['spacing'], 'row': row,
        'presence_probability': float(torch.sigmoid(
            cache['routed']['lesion_presence_logit'].float()
        )[0].cpu()),
        'predicted_volume_ml': float(torch.expm1(
            cache['routed']['log_volume_prediction'].float()
        )[0].clamp_min(0).cpu()),
    }


def pilot_component_tier(volume_ml, voxels):
    if int(voxels) <= 1:
        return 'subvoxel_or_single_voxel'
    if volume_ml < 0.10:
        return 'micro'
    if volume_ml < 1.0:
        return 'tiny'
    if volume_ml < 5.0:
        return 'small'
    if volume_ml < 25.0:
        return 'medium'
    return 'large'


def pilot_lesion_counts(prediction, target):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    if predicted_count == 0 or target_count == 0:
        return 0, int(predicted_count), int(target_count)
    overlap = np.column_stack([
        predicted_labels[prediction & target],
        target_labels[prediction & target],
    ])
    overlap = overlap[(overlap[:, 0] > 0) & (overlap[:, 1] > 0)]
    if len(overlap) == 0:
        return 0, int(predicted_count), int(target_count)
    pairs, counts = np.unique(overlap, axis=0, return_counts=True)
    used_predicted, used_target = set(), set()
    matches = 0
    for index in np.argsort(-counts):
        predicted_id, target_id = [int(v) for v in pairs[index]]
        if predicted_id in used_predicted or target_id in used_target:
            continue
        used_predicted.add(predicted_id)
        used_target.add(target_id)
        matches += 1
    return matches, int(predicted_count - matches), int(target_count - matches)


def pilot_target_component_iou_records(prediction, target, spacing):
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    predicted_labels, predicted_count = ndimage.label(prediction, structure)
    target_labels, target_count = ndimage.label(target, structure)
    predicted_sizes = np.bincount(
        predicted_labels.reshape(-1), minlength=predicted_count + 1
    )
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    records = []
    target_objects = ndimage.find_objects(target_labels)
    for component_id, component_slice in enumerate(target_objects, start=1):
        if component_slice is None:
            continue
        target_local = target_labels[component_slice]
        component_local = target_local == component_id
        component_voxels = int(component_local.sum())
        predicted_local = predicted_labels[component_slice]
        overlapping_ids, intersections = np.unique(
            predicted_local[component_local], return_counts=True
        )
        maximum_iou = 0.0
        matched_prediction_id = 0
        for predicted_id, intersection in zip(overlapping_ids, intersections):
            predicted_id = int(predicted_id)
            if predicted_id == 0:
                continue
            union = (
                component_voxels + int(predicted_sizes[predicted_id])
                - int(intersection)
            )
            iou = float(intersection) / max(union, 1)
            if iou > maximum_iou:
                maximum_iou = iou
                matched_prediction_id = predicted_id
        volume_ml = component_voxels * voxel_volume_ml
        records.append({
            'component_id': component_id,
            'component_voxels': component_voxels,
            'component_volume_ml': volume_ml,
            'component_tier': pilot_component_tier(
                volume_ml, component_voxels
            ),
            'maximum_component_iou': maximum_iou,
            'matched_prediction_id': matched_prediction_id,
        })
    return records


def pilot_threshold_metrics(
    case_cache, threshold, probability_key, include_case_records=False
):
    positive_dice = []
    total_intersection = total_prediction = total_target = 0
    lesion_tp = lesion_fp = lesion_fn = 0
    component_tp = component_total = 0
    small_tp = small_total = 0
    small_case_dice = []
    threshold_case_rows = []
    for case in case_cache:
        row = case['row']
        probability = case[probability_key].astype(np.float32)
        prediction = probability >= float(threshold)
        target = case['target']
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        case_dice = float('nan')
        case_lesion_tp = case_lesion_fp = case_lesion_fn = 0
        case_component_tp = case_component_total = 0
        case_small_tp = case_small_total = 0
        if hard_eligible and target_voxels > 0:
            case_dice = (
                2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            )
            positive_dice.append(case_dice)
            if str(row['trial8_effective_lesion_volume_bin']) in {
                'tiny', 'small'
            }:
                small_case_dice.append(case_dice)
            total_intersection += intersection
            total_prediction += predicted_voxels
            total_target += target_voxels
        if hard_eligible:
            tp, fp, fn = pilot_lesion_counts(prediction, target)
            lesion_tp += tp
            lesion_fp += fp
            lesion_fn += fn
            case_lesion_tp = int(tp)
            case_lesion_fp = int(fp)
            case_lesion_fn = int(fn)
        if hard_eligible and target_voxels > 0:
            iou_threshold = (
                0.05 if probability_key == 'proposal_probability' else 0.10
            )
            for component in pilot_target_component_iou_records(
                prediction, target, case['spacing']
            ):
                tier = component['component_tier']
                detected = (
                    float(component['maximum_component_iou']) >= iou_threshold
                )
                component_total += 1
                component_tp += int(detected)
                case_component_total += 1
                case_component_tp += int(detected)
                if tier in {
                    'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
                }:
                    small_total += 1
                    small_tp += int(detected)
                    case_small_total += 1
                    case_small_tp += int(detected)
        if include_case_records:
            voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
            threshold_case_rows.append({
                'subject_id': str(case['subject_id']),
                'probability_source': str(probability_key),
                'threshold': float(threshold),
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'hard_metric_eligible': bool(hard_eligible),
                'target_voxels': int(target_voxels),
                'predicted_voxels': int(predicted_voxels),
                'intersection_voxels': int(intersection),
                'case_dice': float(case_dice),
                'target_volume_ml': float(target_voxels * voxel_volume_ml),
                'predicted_volume_ml': float(
                    predicted_voxels * voxel_volume_ml
                ),
                'lesion_true_positive': int(case_lesion_tp),
                'lesion_false_positive': int(case_lesion_fp),
                'lesion_false_negative': int(case_lesion_fn),
                'component_true_positive': int(case_component_tp),
                'component_total': int(case_component_total),
                'small_component_true_positive': int(case_small_tp),
                'small_component_total': int(case_small_total),
            })
    lesion_precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    lesion_recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * lesion_precision * lesion_recall
        / max(lesion_precision + lesion_recall, 1e-12)
    )
    mean_dice = float(np.mean(positive_dice)) if positive_dice else 0.0
    global_dice = float(
        2.0 * total_intersection / max(total_prediction + total_target, 1)
    )
    component_recall = component_tp / max(component_total, 1)
    small_recall = small_tp / max(small_total, 1)
    small_mean_dice = (
        float(np.mean(small_case_dice)) if small_case_dice else 0.0
    )
    selection_score = (
        0.60 * mean_dice + 0.20 * global_dice
        + 0.15 * lesion_f1 + 0.05 * small_mean_dice
    )
    summary = {
        'threshold': float(threshold),
        'positive_mean_dice': mean_dice,
        'positive_median_dice': float(np.median(positive_dice)) if positive_dice else 0.0,
        'positive_global_dice': global_dice,
        'lesion_precision': float(lesion_precision),
        'lesion_recall': float(lesion_recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': float(component_recall),
        'small_component_recall': float(small_recall),
        'small_case_mean_dice': float(small_mean_dice),
        'selection_score': float(selection_score),
        'positive_subjects': len(positive_dice),
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }
    if include_case_records:
        return summary, threshold_case_rows
    return summary


@torch.no_grad()
def pilot_validate(model, epoch, device):
    model.eval()
    case_cache = []
    for index, row in enumerate(heldout_rows, start=1):
        case_cache.append(pilot_infer_case(model, row, device))
        print(
            f'  Fold-{REPLICATION_FOLD} held-out inference '
            f'{index:02d}/{len(heldout_rows)}: '
            f"{row['subject_id']}"
        )
    local_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'probability', include_case_records=True
        )
        for threshold in LOCAL_THRESHOLDS
    ]
    local_threshold_df = pd.DataFrame([
        result[0] for result in local_threshold_results
    ]).sort_values(
        ['selection_score', 'positive_mean_dice', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    local_threshold_case_df = pd.DataFrame([
        row
        for _, rows in local_threshold_results
        for row in rows
    ])
    local_threshold_case_df.insert(0, 'epoch', int(epoch))
    local_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    proposal_threshold_results = [
        pilot_threshold_metrics(
            case_cache, threshold, 'proposal_probability',
            include_case_records=True,
        )
        for threshold in PROPOSAL_THRESHOLDS
    ]
    proposal_threshold_df = pd.DataFrame([
        result[0] for result in proposal_threshold_results
    ])
    proposal_threshold_case_df = pd.DataFrame([
        row
        for _, rows in proposal_threshold_results
        for row in rows
    ])
    proposal_threshold_case_df.insert(0, 'epoch', int(epoch))
    proposal_threshold_case_df.insert(
        0, 'development_fold', int(REPLICATION_FOLD)
    )
    expected_local_rows = len(heldout_rows) * len(LOCAL_THRESHOLDS)
    expected_proposal_rows = len(heldout_rows) * len(PROPOSAL_THRESHOLDS)
    if len(local_threshold_case_df) != expected_local_rows:
        raise RuntimeError(
            'Incomplete local OOF threshold-case audit: '
            f'{len(local_threshold_case_df)} != {expected_local_rows}.'
        )
    if len(proposal_threshold_case_df) != expected_proposal_rows:
        raise RuntimeError(
            'Incomplete proposal OOF threshold-case audit: '
            f'{len(proposal_threshold_case_df)} != {expected_proposal_rows}.'
        )
    proposal_threshold_df['proposal_selection_score'] = (
        0.45 * proposal_threshold_df['positive_global_dice']
        + 0.25 * proposal_threshold_df['lesion_f1']
        + 0.15 * proposal_threshold_df['lesion_precision']
        + 0.15 * proposal_threshold_df['component_recall']
    )
    proposal_threshold_df = proposal_threshold_df.sort_values(
        ['proposal_selection_score', 'component_recall', 'threshold'],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    selected = local_threshold_df.iloc[0].to_dict()
    proposal_selected = proposal_threshold_df.iloc[0].to_dict()
    local_threshold = float(selected['threshold'])
    proposal_threshold = float(proposal_selected['threshold'])

    case_rows = []
    component_rows = []
    tier_counts = defaultdict(lambda: [0, 0, 0])
    lesion_bin_dice = defaultdict(list)
    presence_squared_errors = []
    router_volume_errors_ml = []
    empty_false_positive_volumes_ml = []
    representation_failures = proposal_failures = refinement_failures = 0
    for case in case_cache:
        row = case['row']
        probability = case['probability'].astype(np.float32)
        proposal = case['proposal_probability'].astype(np.float32)
        target = case['target']
        prediction = probability >= local_threshold
        proposal_prediction = proposal >= proposal_threshold
        intersection = int((prediction & target).sum())
        predicted_voxels = int(prediction.sum())
        target_voxels = int(target.sum())
        dice = (
            2.0 * intersection / max(predicted_voxels + target_voxels, 1)
            if predicted_voxels + target_voxels > 0 else 1.0
        )
        voxel_volume_ml = float(np.prod(case['spacing']) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        if hard_eligible and target_voxels > 0:
            lesion_bin_dice[str(
                row['trial8_effective_lesion_volume_bin']
            )].append(float(dice))
        if hard_eligible:
            presence_target = float(target_voxels > 0)
            presence_squared_errors.append(
                (case['presence_probability'] - presence_target) ** 2
            )
            target_volume_ml = target_voxels * voxel_volume_ml
            router_volume_errors_ml.append(abs(
                case['predicted_volume_ml'] - target_volume_ml
            ))
            if target_voxels == 0:
                empty_false_positive_volumes_ml.append(
                    predicted_voxels * voxel_volume_ml
                )
        case_rows.append({
            'epoch': int(epoch), 'subject_id': case['subject_id'],
            'lesion_volume_bin': str(row['trial8_effective_lesion_volume_bin']),
            'hard_metric_eligible': as_bool(row['trial8_hard_metric_eligible']),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': target_voxels,
            'predicted_voxels': predicted_voxels,
            'dice': float(dice),
            'voxel_precision': intersection / max(predicted_voxels, 1),
            'voxel_recall': intersection / max(target_voxels, 1),
            'target_volume_ml': target_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_mask': predicted_voxels * voxel_volume_ml,
            'predicted_volume_ml_from_router': case['predicted_volume_ml'],
            'presence_probability': case['presence_probability'],
        })
        if hard_eligible and target_voxels > 0:
            labels, count = ndimage.label(
                target, np.ones((3, 3, 3), dtype=np.uint8)
            )
            final_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    prediction, target, case['spacing']
                )
            }
            proposal_iou = {
                int(item['component_id']): item
                for item in pilot_target_component_iou_records(
                    proposal_prediction, target, case['spacing']
                )
            }
            component_objects = ndimage.find_objects(labels)
            for component_id, component_slice in enumerate(
                component_objects, start=1
            ):
                if component_slice is None:
                    continue
                component_local = (
                    labels[component_slice] == component_id
                )
                voxels = int(component_local.sum())
                volume_ml = voxels * voxel_volume_ml
                tier = pilot_component_tier(volume_ml, voxels)
                final_max = float(
                    probability[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                proposal_max = float(
                    proposal[component_slice][component_local].max(
                        initial=0.0
                    )
                )
                final_max_iou = float(
                    final_iou[component_id]['maximum_component_iou']
                )
                proposal_max_iou = float(
                    proposal_iou[component_id]['maximum_component_iou']
                )
                final_detected = final_max_iou >= 0.10
                proposal_covered = proposal_max_iou >= 0.05
                representation_failure = final_max < 0.10
                if representation_failure:
                    representation_failures += 1
                if not proposal_covered:
                    failure_class = 'proposal_failure'
                    proposal_failures += 1
                elif not final_detected:
                    failure_class = 'refinement_failure'
                    refinement_failures += 1
                else:
                    failure_class = 'detected'
                tier_counts[tier][0] += int(final_detected)
                tier_counts[tier][1] += int(proposal_covered)
                tier_counts[tier][2] += 1
                component_rows.append({
                    'epoch': int(epoch), 'subject_id': case['subject_id'],
                    'component_id': component_id, 'component_tier': tier,
                    'component_voxels': voxels,
                    'component_volume_ml': volume_ml,
                    'final_probability_maximum': final_max,
                    'proposal_probability_maximum': proposal_max,
                    'final_maximum_component_iou': final_max_iou,
                    'proposal_maximum_component_iou': proposal_max_iou,
                    'final_detected': bool(final_detected),
                    'proposal_covered': bool(proposal_covered),
                    'representation_failure_below_0p10': representation_failure,
                    'failure_class': failure_class,
                })
    tier_rows = []
    for tier in COMPONENT_TIERS:
        final_tp, proposal_tp, total = tier_counts[tier]
        tier_rows.append({
            'epoch': int(epoch), 'component_tier': tier,
            'components': total,
            'final_component_recall': final_tp / max(total, 1),
            'proposal_component_recall': proposal_tp / max(total, 1),
        })
    pilot_score = (
        0.90 * float(selected['selection_score'])
        + 0.10 * float(proposal_selected['positive_global_dice'])
    )
    summary = {
        **{key: to_builtin(value) for key, value in selected.items()},
        'proposal_threshold': proposal_threshold,
        'proposal_component_recall': float(
            proposal_selected['component_recall']
        ),
        'proposal_global_dice': float(
            proposal_selected['positive_global_dice']
        ),
        'proposal_lesion_precision': float(
            proposal_selected['lesion_precision']
        ),
        'pilot_checkpoint_score': float(pilot_score),
        'representation_failures': int(representation_failures),
        'proposal_failures': int(proposal_failures),
        'refinement_failures': int(refinement_failures),
        'components_audited': len(component_rows),
        'presence_brier_score': float(
            np.mean(presence_squared_errors)
            if presence_squared_errors else float('nan')
        ),
        'router_volume_mae_ml': float(
            np.mean(router_volume_errors_ml)
            if router_volume_errors_ml else float('nan')
        ),
        'empty_false_positive_volume_ml': float(
            np.mean(empty_false_positive_volumes_ml)
            if empty_false_positive_volumes_ml else 0.0
        ),
        'lesion_bin_mean_dice': {
            name: float(np.mean(values)) if values else float('nan')
            for name, values in lesion_bin_dice.items()
        },
    }
    epoch_root = PILOT_VALIDATION_ROOT / f'epoch_{int(epoch):03d}'
    epoch_root.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(local_threshold_df, epoch_root / 'local_threshold_summary.csv')
    atomic_write_csv(
        local_threshold_case_df,
        epoch_root / 'local_threshold_case_metrics.csv',
    )
    atomic_write_csv(
        proposal_threshold_df, epoch_root / 'proposal_threshold_summary.csv'
    )
    atomic_write_csv(
        proposal_threshold_case_df,
        epoch_root / 'proposal_threshold_case_metrics.csv',
    )
    atomic_write_csv(pd.DataFrame(case_rows), epoch_root / 'case_metrics.csv')
    atomic_write_csv(
        pd.DataFrame(component_rows), epoch_root / 'component_metrics.csv'
    )
    atomic_write_csv(
        pd.DataFrame(tier_rows), epoch_root / 'component_tier_metrics.csv'
    )
    atomic_write_json(summary, epoch_root / 'validation_summary.json')
    del case_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, tier_rows






def pilot_branch_subjects():
    frame = active_heldout_df.copy()
    preferred = []
    for lesion_bin in ['tiny', 'small', 'medium', 'very_large']:
        rows = frame[
            frame['trial8_effective_lesion_volume_bin'].astype(str) == lesion_bin
        ]
        if len(rows):
            preferred.append(rows.iloc[0].to_dict())
    unique = []
    seen = set()
    for row in preferred:
        subject_id = str(row['subject_id'])
        if subject_id not in seen:
            unique.append(row)
            seen.add(subject_id)
        if len(unique) == 3:
            break
    if len(unique) < 3:
        for row in heldout_rows:
            if str(row['subject_id']) not in seen:
                unique.append(row)
                seen.add(str(row['subject_id']))
            if len(unique) == 3:
                break
    return unique


BRANCH_PROBE_ROWS = pilot_branch_subjects()


def pilot_branch_probe_origin(case):
    target = case['target']
    labels, count = ndimage.label(
        target, np.ones((3, 3, 3), dtype=np.uint8)
    )
    if count == 0:
        center = np.asarray(target.shape) // 2
    else:
        components = []
        for component_id in range(1, count + 1):
            coordinates = np.argwhere(labels == component_id)
            components.append((len(coordinates), coordinates.mean(axis=0)))
        components.sort(key=lambda item: item[0])
        center = components[0][1]
    return pilot_origin_from_center(center, target.shape)


@torch.no_grad()
def pilot_branch_reliance(model, epoch, device):
    model.eval()
    rows = []
    ablations = ['global', 'arterial', 'clinical', 'occlusion', 'vascular', 'atlas']
    for row in BRANCH_PROBE_ROWS:
        case = pilot_load_validation_case(row)
        origin = pilot_branch_probe_origin(case)
        sample = pilot_window_sample(case, origin)
        batch = pilot_stack_sample(sample, device)
        binary_canvas, _ = symmetric_pad(
            case['target'].astype(np.uint8), GLOBAL_CANVAS_SHAPE, (0, 1, 2)
        )
        local_target = crop_at(
            binary_canvas, origin, LOCAL_PATCH_SIZE, (0, 1, 2)
        ).astype(bool)

        def run(ablation):
            force_atlas = ablation == 'atlas'
            force_arterial = ablation in {'arterial', 'atlas'}
            force_clinical = ablation == 'clinical'
            force_occlusion = ablation == 'occlusion'
            with torch.autocast(
                device_type=device.type, dtype=torch.bfloat16,
                enabled=device.type == 'cuda',
            ):
                cache = pilot_global_cache(
                    model, batch, force_atlas_dropout=force_atlas,
                    force_clinical_dropout=force_clinical,
                    force_occlusion_dropout=force_occlusion,
                    force_arterial_dropout=force_arterial,
                )
                output = pilot_local_from_cache(
                    model, batch, cache,
                    force_global_dropout=(ablation == 'global'),
                    force_atlas_dropout=force_atlas,
                    force_vascular_dropout=(ablation == 'vascular'),
                    force_occlusion_dropout=force_occlusion,
                    force_clinical_dropout=force_clinical,
                )
            return (
                torch.sigmoid(output['logits'].float())[0, 0].cpu().numpy(),
                torch.sigmoid(cache['proposal_logits'].float())[0, 0].cpu().numpy(),
            )

        baseline_local, baseline_proposal = run('none')
        for ablation in ablations:
            ablated_local, ablated_proposal = run(ablation)
            target_delta = (
                float(np.mean(np.abs(
                    baseline_local[local_target] - ablated_local[local_target]
                ))) if bool(local_target.any()) else float('nan')
            )
            rows.append({
                'epoch': int(epoch), 'subject_id': case['subject_id'],
                'lesion_volume_bin': str(
                    row['trial8_effective_lesion_volume_bin']
                ),
                'ablation': ablation,
                'mean_absolute_local_probability_delta': float(
                    np.mean(np.abs(baseline_local - ablated_local))
                ),
                'maximum_absolute_local_probability_delta': float(
                    np.max(np.abs(baseline_local - ablated_local))
                ),
                'mean_absolute_target_probability_delta': target_delta,
                'mean_absolute_proposal_probability_delta': float(
                    np.mean(np.abs(baseline_proposal - ablated_proposal))
                ),
                'baseline_target_probability_maximum': float(
                    baseline_local[local_target].max(initial=0.0)
                ),
                'ablated_target_probability_maximum': float(
                    ablated_local[local_target].max(initial=0.0)
                ),
            })
    frame = pd.DataFrame(rows)
    atomic_write_csv(
        frame, PILOT_DIAGNOSTIC_ROOT / f'epoch_{int(epoch):03d}_branch_reliance.csv'
    )
    if PILOT_BRANCH_HISTORY_PATH.exists():
        prior = pd.read_csv(PILOT_BRANCH_HISTORY_PATH)
        prior = prior[prior['epoch'].astype(int) != int(epoch)]
        frame = pd.concat([prior, frame], ignore_index=True)
    atomic_write_csv(frame, PILOT_BRANCH_HISTORY_PATH)
    return frame[frame['epoch'].astype(int) == int(epoch)].copy()






def pilot_activate_bounded_gates(model):
    fraction = 0.15
    magnitude = math.atanh(fraction)
    confidence_gates = [
        model.local_model.vascular_fusion_quarter.gate,
        model.local_model.vascular_fusion_eighth.gate,
        model.local_model.vascular_fusion_bottleneck.gate,
        model.local_model.context_fusion.gate,
        model.local_fusion.gate,
    ]
    with torch.no_grad():
        for gate in confidence_gates:
            gate.network[-1].weight.zero_()
            gate.network[-1].bias.fill_(magnitude)
        final = model.global_stream.context_film.project[-1]
        final.weight.zero_()
        final.bias.zero_()
        rows = torch.arange(final.out_features, device=final.weight.device)
        columns = rows.remainder(final.in_features)
        final.weight[rows, columns] = math.atanh(0.10)


def pilot_parameter_groups(model):
    groups = {'preserved': [], 'new': [], 'gate': []}
    seen = set()
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        identity = id(parameter)
        if identity in seen:
            raise RuntimeError(f'Duplicate optimizer parameter: {name}')
        seen.add(identity)
        is_gate = (
            '.gate.network.2.' in name
            or 'global_stream.context_film.project.2.' in name
        )
        is_new = name.startswith((
            'arterial_projector.', 'global_stream.', 'global_router.',
            'local_fusion.',
        ))
        groups['gate' if is_gate else ('new' if is_new else 'preserved')].append(
            parameter
        )
    if len(seen) != sum(1 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Optimizer parameter partition is incomplete.')
    return [
        {'params': groups['preserved'], 'lr': BASE_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'preserved_v2'},
        {'params': groups['new'], 'lr': NEW_PATH_LR, 'weight_decay': WEIGHT_DECAY,
         'group_name': 'new_v2_2'},
        {'params': groups['gate'], 'lr': GATE_LR, 'weight_decay': 0.0,
         'group_name': 'bounded_gates'},
    ], {name: sum(p.numel() for p in values) for name, values in groups.items()}


def pilot_update_ema(ema_model, model, update_step):
    decay = min(EMA_MAX_DECAY, 1.0 - 1.0 / float(update_step + 1))
    with torch.no_grad():
        source = model.state_dict()
        for name, value in ema_model.state_dict().items():
            if value.dtype.is_floating_point:
                value.mul_(decay).add_(source[name].detach(), alpha=1.0 - decay)
            else:
                value.copy_(source[name])


def pilot_config_hash(config):
    payload = json.dumps(
        to_builtin(config), sort_keys=True, separators=(',', ':'), allow_nan=False
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def pilot_recompute_selection_state(history_rows):
    
    if not history_rows:
        return {
            'last_epoch': 0,
            'numeric_best_score': -float('inf'),
            'numeric_best_epoch': 0,
            'patience_reference_score': -float('inf'),
            'patience_reference_epoch': 0,
            'validations_without_meaningful_improvement': 0,
        }
    ordered = sorted(history_rows, key=lambda row: int(row['epoch']))
    epochs = [int(row['epoch']) for row in ordered]
    expected_epochs = list(range(1, epochs[-1] + 1))
    if epochs != expected_epochs:
        raise RuntimeError(
            'Training history is not a unique contiguous epoch sequence: '
            f'observed={epochs}, expected={expected_epochs}.'
        )
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    for row in ordered:
        epoch = int(row['epoch'])
        score = float(row['pilot_checkpoint_score'])
        if not math.isfinite(score):
            raise RuntimeError(
                f'Epoch {epoch}: history contains a non-finite checkpoint score.'
            )
        numeric_improved = score > numeric_best_score
        if numeric_improved:
            numeric_best_score = score
            numeric_best_epoch = epoch
        meaningful = (
            score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if meaningful:
            patience_reference_score = score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        row_state_checks = {
            'numeric_best_score': (
                'numeric_best_score' not in row
                or pilot_float_matches(
                    row['numeric_best_score'], numeric_best_score
                )
            ),
            'numeric_best_epoch': (
                'numeric_best_epoch' not in row
                or int(row['numeric_best_epoch']) == numeric_best_epoch
            ),
            'numeric_best_improved': (
                'numeric_best_improved' not in row
                or as_bool(row['numeric_best_improved']) == numeric_improved
            ),
            'patience_reference_score': (
                'patience_reference_score' not in row
                or pilot_float_matches(
                    row['patience_reference_score'],
                    patience_reference_score,
                )
            ),
            'patience_reference_epoch': (
                'patience_reference_epoch' not in row
                or int(row['patience_reference_epoch'])
                == patience_reference_epoch
            ),
            'patience_meaningfully_improved': (
                'patience_meaningfully_improved' not in row
                or as_bool(row['patience_meaningfully_improved'])
                == meaningful
            ),
            'patience_counter': (
                'validations_without_meaningful_improvement' not in row
                or int(
                    row['validations_without_meaningful_improvement']
                ) == validations_without_meaningful_improvement
            ),
        }
        if not all(row_state_checks.values()):
            failed = [
                name for name, passed in row_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                f'Epoch {epoch}: committed history selection state is '
                'inconsistent: ' + ', '.join(failed)
            )
    return {
        'last_epoch': epochs[-1],
        'numeric_best_score': numeric_best_score,
        'numeric_best_epoch': numeric_best_epoch,
        'patience_reference_score': patience_reference_score,
        'patience_reference_epoch': patience_reference_epoch,
        'validations_without_meaningful_improvement': (
            validations_without_meaningful_improvement
        ),
    }


def pilot_float_matches(left, right, atol=STRICT_BEST_SCORE_ATOL):
    return math.isclose(
        float(left), float(right), rel_tol=0.0, abs_tol=float(atol)
    )


def pilot_diagnostic_interpretation(summary, tier_rows, branch_frame):
    tier = {row['component_tier']: row for row in tier_rows}
    small_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[:4]
        if name in tier and int(tier[name]['components']) > 0
    ]
    large_values = [
        tier[name]['final_component_recall'] for name in COMPONENT_TIERS[4:]
        if name in tier and int(tier[name]['components']) > 0
    ]
    small_recall = float(np.mean(small_values)) if small_values else 0.0
    large_recall = float(np.mean(large_values)) if large_values else 0.0
    branch_summary = {}
    if len(branch_frame):
        branch_summary = branch_frame.groupby('ablation')[
            'mean_absolute_target_probability_delta'
        ].mean().fillna(0.0).to_dict()
    proposal_recall = float(summary['proposal_component_recall'])
    final_recall = float(summary['component_recall'])
    findings = {
        'global_proposal_bottleneck': proposal_recall < 0.60,
        'local_refinement_bottleneck': (
            proposal_recall - final_recall >= 0.15
        ),
        'small_component_generalization_gap': (
            large_recall > 0 and small_recall < 0.50 * large_recall
        ),
        'representation_failure_dominant': (
            int(summary['representation_failures'])
            > 0.50 * max(int(summary['components_audited']), 1)
        ),
        'clinical_branch_weakly_used': float(
            branch_summary.get('clinical', 0.0)
        ) < 0.002,
        'arterial_branch_weakly_used': float(
            branch_summary.get('arterial', 0.0)
        ) < 0.002,
    }
    if findings['global_proposal_bottleneck']:
        recommendation = (
            'Improve proposal supervision/resolution before adding more routing.'
        )
    elif findings['local_refinement_bottleneck']:
        recommendation = (
            'Keep the global proposal and strengthen proposal-to-local refinement.'
        )
    elif findings['small_component_generalization_gap']:
        recommendation = (
            'Retain the architecture and increase micro/tiny component curriculum.'
        )
    elif findings['clinical_branch_weakly_used']:
        recommendation = (
            'Audit pretreatment clinical signal and context-gate optimization.'
        )
    else:
        recommendation = (
            'Architecture is functioning; proceed only after manual pilot review.'
        )
    return findings, branch_summary, recommendation


def run_v22_replication_fold():
    if ALL_REPLICATION_FOLDS_COMPLETE:
        series_status = write_final_series_status()
        print('\n' + '=' * 80)
        print('TRIAL8 V2.2 R5 FINAL FIVE-FOLD SERIES ALREADY COMPLETE')
        print('=' * 80)
        print(f'Completed folds:                     {COMPLETED_REPLICATION_FOLDS}')
        print(
            'Out-of-fold subjects:                '
            f'{series_status["completed_heldout_subjects"]}'
        )
        print(f'Series root:                         {REPLICATION_SERIES_ROOT}')
        print(f'Series status:                       {FINAL_SERIES_STATUS_PATH}')
        print('No model, optimizer, checkpoint, or history was changed.')
        return

    if PILOT_RESULT_GATE_PATH.exists():
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has an existing result gate that failed '
            'the final artifact-integrity contract. Refusing to skip, resume, '
            'or overwrite this fold. Preserve the directory and diagnose the '
            f'gate before continuing: {PILOT_RESULT_GATE_PATH}'
        )

    set_determinism(PILOT_SEED)
    if not torch.cuda.is_available():
        raise RuntimeError('Trial8 V2.2 replication requires a CUDA GPU.')
    gc.collect()
    torch.cuda.empty_cache()
    device = torch.device('cuda')
    model = VascuFateNetV22(pilot_schema).to(device)
    if sum(p.numel() for p in model.parameters()) != EXPECTED_V22_PARAMETERS:
        raise RuntimeError('V2.2 parameter count changed before training.')
    pilot_activate_bounded_gates(model)
    ema_model = deepcopy(model).to(device).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    dataset = Trial8V22ReplicationDataset()
    smoke_index = None
    for index, entry in enumerate(dataset.schedule):
        row = fit_row_by_subject[str(entry['subject_id'])]
        if not as_bool(row['trial8_atlas_enabled']):
            continue
        with np.load(
            Path(str(row['trial8_graph_sidecar_path'])), allow_pickle=False
        ) as sidecar:
            vessel_present = np.asarray(sidecar['vessel_presence']) > 0
        if bool(vessel_present.any()):
            smoke_index = int(index)
            break
    if smoke_index is None:
        raise RuntimeError(
            f'Fold-{REPLICATION_FOLD} has no atlas/vessel-valid fitting '
            'sample for the complete-path gradient smoke test.'
        )
    smoke_loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, [smoke_index]),
        batch_size=1, shuffle=False, num_workers=0,
        pin_memory=True,
    )
    smoke_batch = next(iter(smoke_loader))
    model.train()
    with torch.autocast(
        device_type='cuda', dtype=torch.bfloat16, enabled=True
    ):
        smoke_loss, smoke_parts = pilot_compute_training_loss(
            model, smoke_batch, device, epoch=1, batch_index=0
        )
    if not bool(torch.isfinite(smoke_loss).item()):
        raise RuntimeError('V2.2 replication smoke loss is non-finite.')
    smoke_loss.backward()
    gradient_groups = {
        'preserved_local': 'local_model.',
        'global_proposal': 'global_stream.',
        'arterial_projector': 'arterial_projector.',
        'global_router': 'global_router.',
        'local_fusion': 'local_fusion.',
    }
    gradient_checks = {}
    for label, prefix in gradient_groups.items():
        gradients = [
            p.grad for name, p in model.named_parameters()
            if name.startswith(prefix) and p.grad is not None
        ]
        gradient_checks[label] = bool(gradients) and all(
            bool(torch.isfinite(g).all().item()) for g in gradients
        ) and any(float(g.detach().float().norm().cpu()) > 0 for g in gradients)
    model.zero_grad(set_to_none=True)
    if not all(gradient_checks.values()):
        failed = [name for name, passed in gradient_checks.items() if not passed]
        raise RuntimeError('Pilot gradient smoke failed: ' + ', '.join(failed))

    parameter_groups, group_counts = pilot_parameter_groups(model)
    optimizer = torch.optim.AdamW(
        parameter_groups, betas=(0.9, 0.999), eps=1e-8
    )
    updates_per_epoch = int(math.ceil(len(dataset) / ACCUMULATION_STEPS))
    total_updates = updates_per_epoch * MAX_EPOCHS
    warmup_updates = updates_per_epoch * 2

    def lr_lambda(update):
        if update < warmup_updates:
            return max((update + 1) / max(warmup_updates, 1), 0.05)
        progress = (update - warmup_updates) / max(
            total_updates - warmup_updates, 1
        )
        return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    pilot_config = {
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'architecture': MODEL_VERSION,
        'fold': REPLICATION_FOLD,
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'local_patch_size': list(LOCAL_PATCH_SIZE),
        'global_canvas_shape': list(GLOBAL_CANVAS_SHAPE),
        'global_input_shape': list(GLOBAL_INPUT_SHAPE),
        'samples_per_epoch': len(dataset),
        'positive_patches': POSITIVE_PATCHES,
        'topology_negative_patches': TOPOLOGY_NEGATIVE_PATCHES,
        'brain_negative_patches': BRAIN_NEGATIVE_PATCHES,
        'maximum_epochs': MAX_EPOCHS,
        'minimum_epochs': MINIMUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_selection_policy': (
            'strict_numerical_maximum_of_frozen_composite_score'
        ),
        'patience_policy': (
            'reset_only_when_score_exceeds_patience_reference_by_0p002; '
            'minimum_epoch_20; patience_10'
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'accumulation_steps': ACCUMULATION_STEPS,
        'base_lr': BASE_LR, 'new_path_lr': NEW_PATH_LR,
        'gate_lr': GATE_LR, 'weight_decay': WEIGHT_DECAY,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'ema_max_decay': EMA_MAX_DECAY,
        'local_thresholds': LOCAL_THRESHOLDS.tolist(),
        'proposal_thresholds': PROPOSAL_THRESHOLDS.tolist(),
        'checkpoint_score': (
            '0.90*(0.60*mean_dice+0.20*global_dice+'
            '0.15*lesion_f1+0.05*small_case_mean_dice)+'
            '0.10*proposal_global_dice'
        ),
        'component_detection_policy': (
            'final IoU >= 0.10; global proposal IoU >= 0.05'
        ),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'same_run_resume_policy': 'same_fold_last_checkpoint_only',
        'r5_source_config_sha256': file_sha256(R5_SOURCE_CONFIG_PATH),
        'r5_source_result_gate_sha256': file_sha256(
            R5_SOURCE_RESULT_GATE_PATH
        ),
        'fivefold_authorization_gate_sha256': file_sha256(
            AUTHORIZATION_GATE_PATH
        ),
        'graph_scaler_path': str(ACTIVE_GRAPH_SCALER_PATH),
        'graph_scaler_sha256': file_sha256(ACTIVE_GRAPH_SCALER_PATH),
        'clinical_scaler_path': str(ACTIVE_CLINICAL_SCALER_PATH),
        'clinical_scaler_sha256': file_sha256(ACTIVE_CLINICAL_SCALER_PATH),
        'optimizer_parameter_counts': group_counts,
        'input_sha256': pilot_input_hashes,
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'development_fivefold_replication_authorized': True,
        'active_fold_training_authorized': True,
        'active_fold_in_final_protocol': (
            REPLICATION_FOLD in AUTHORIZED_REPLICATION_FOLDS
        ),
    }
    config_hash = pilot_config_hash(pilot_config)
    if PILOT_CONFIG_PATH.exists():
        existing = load_json(PILOT_CONFIG_PATH)
        if pilot_config_hash(existing) != config_hash:
            raise RuntimeError(
                'Existing V2.2 replication configuration differs. Refusing '
                'overwrite.'
            )
    else:
        atomic_write_json(pilot_config, PILOT_CONFIG_PATH)
    start_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'upstream_checks': pilot_upstream_checks,
        'target_access_checks': pilot_target_access_checks,
        'gradient_checks': gradient_checks,
        'gradient_smoke_schedule_index': smoke_index,
        'gradient_smoke_subject_id': str(
            dataset.schedule[smoke_index]['subject_id']
        ),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'smoke_loss': float(smoke_loss.detach().cpu()),
        'fresh_start_from_initialization': FRESH_START_FROM_INITIALIZATION,
        'parent_checkpoint_migration_allowed': (
            PARENT_CHECKPOINT_MIGRATION_ALLOWED
        ),
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_checkpointing': True,
        'patience_reference_is_independent': True,
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
    }
    atomic_write_json(start_gate, PILOT_START_GATE_PATH)

    start_epoch = 1
    update_step = 0
    numeric_best_score = -float('inf')
    numeric_best_epoch = 0
    numeric_best_threshold = 0.50
    numeric_best_proposal_threshold = 0.10
    patience_reference_score = -float('inf')
    patience_reference_epoch = 0
    validations_without_meaningful_improvement = 0
    resume_stop_already_triggered = False
    resume_discarded_uncommitted_history_epochs = []
    resume_recovered_strict_best_from_last = False
    history_rows = []
    if PILOT_LAST_CHECKPOINT_PATH.exists():
        checkpoint = torch.load(
            PILOT_LAST_CHECKPOINT_PATH, map_location=device,
            weights_only=False,
        )
        resume_checks = {
            'pilot_version': checkpoint.get('pilot_version') == PILOT_VERSION,
            'code_revision': checkpoint.get('code_revision') == PILOT_CODE_REVISION,
            'replication_series': (
                checkpoint.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
            ),
            'development_fold': (
                checkpoint.get('development_fold') == REPLICATION_FOLD
            ),
            'config_hash': checkpoint.get('config_hash') == config_hash,
            'parameters': int(checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS,
            'fresh_start_lineage': (
                checkpoint.get('fresh_start_from_initialization') is True
            ),
            'parent_migration_disabled': (
                checkpoint.get('parent_checkpoint_migration_allowed') is False
            ),
            'strict_best_policy': (
                checkpoint.get('checkpoint_selection_policy')
                == 'strict_numerical_maximum'
            ),
            'independent_patience_policy': (
                checkpoint.get('checkpoint_and_patience_state_are_independent')
                is True
            ),
        }
        if not all(resume_checks.values()):
            failed = [k for k, v in resume_checks.items() if not v]
            raise RuntimeError('Unsafe checkpoint resume: ' + ', '.join(failed))
        model.load_state_dict(checkpoint['model_state'])
        ema_model.load_state_dict(checkpoint['ema_model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        pilot_restore_rng_state(checkpoint['rng_state'])
        checkpoint_epoch = int(checkpoint['epoch'])
        start_epoch = checkpoint_epoch + 1
        update_step = int(checkpoint['update_step'])
        numeric_best_score = float(checkpoint['numeric_best_score'])
        numeric_best_epoch = int(checkpoint['numeric_best_epoch'])
        numeric_best_threshold = float(
            checkpoint['numeric_best_threshold']
        )
        numeric_best_proposal_threshold = float(
            checkpoint['numeric_best_proposal_threshold']
        )
        patience_reference_score = float(
            checkpoint['patience_reference_score']
        )
        patience_reference_epoch = int(
            checkpoint['patience_reference_epoch']
        )
        validations_without_meaningful_improvement = int(
            checkpoint['validations_without_meaningful_improvement']
        )
        if not PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'Last checkpoint exists but its committed training history is '
                'missing; refusing an unverifiable resume.'
            )
        loaded_history_rows = pd.read_csv(
            PILOT_HISTORY_PATH
        ).to_dict('records')
        resume_discarded_uncommitted_history_epochs = sorted({
            int(row['epoch']) for row in loaded_history_rows
            if int(row['epoch']) > checkpoint_epoch
        })
        history_rows = [
            row for row in loaded_history_rows
            if int(row['epoch']) <= checkpoint_epoch
        ]
        reconstructed = pilot_recompute_selection_state(history_rows)
        resume_state_checks = {
            'history_ends_at_checkpoint_epoch': (
                reconstructed['last_epoch'] == checkpoint_epoch
            ),
            'numeric_best_score_matches_history': pilot_float_matches(
                numeric_best_score, reconstructed['numeric_best_score']
            ),
            'numeric_best_epoch_matches_history': (
                numeric_best_epoch == reconstructed['numeric_best_epoch']
            ),
            'patience_reference_score_matches_history': pilot_float_matches(
                patience_reference_score,
                reconstructed['patience_reference_score'],
            ),
            'patience_reference_epoch_matches_history': (
                patience_reference_epoch
                == reconstructed['patience_reference_epoch']
            ),
            'patience_counter_matches_history': (
                validations_without_meaningful_improvement
                == reconstructed[
                    'validations_without_meaningful_improvement'
                ]
            ),
        }
        if not all(resume_state_checks.values()):
            failed = [
                name for name, passed in resume_state_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Checkpoint/history selection-state audit failed: '
                + ', '.join(failed)
            )
        last_record_can_recover_strict_best = bool(
            checkpoint_epoch == numeric_best_epoch
            and pilot_float_matches(
                checkpoint.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        committed_best = None
        if PILOT_BEST_CHECKPOINT_PATH.exists():
            try:
                committed_best = torch.load(
                    PILOT_BEST_CHECKPOINT_PATH,
                    map_location='cpu',
                    weights_only=False,
                )
            except Exception:
                committed_best = None
        committed_best_matches = bool(
            committed_best is not None
            and committed_best.get('pilot_version') == PILOT_VERSION
            and committed_best.get('code_revision') == PILOT_CODE_REVISION
            and committed_best.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and committed_best.get('development_fold') == REPLICATION_FOLD
            and committed_best.get('config_hash') == config_hash
            and int(committed_best.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
            and int(committed_best.get('epoch', -1)) == numeric_best_epoch
            and int(committed_best.get('numeric_best_epoch', -1))
            == numeric_best_epoch
            and pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            )
            and pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            )
        )
        if not committed_best_matches:
            if not last_record_can_recover_strict_best:
                raise RuntimeError(
                    'Strict-best checkpoint is missing or inconsistent, and '
                    'the committed last checkpoint is not the numerical-best '
                    'epoch; automatic recovery is impossible.'
                )
            pilot_atomic_torch_save(checkpoint, PILOT_BEST_CHECKPOINT_PATH)
            resume_recovered_strict_best_from_last = True
            if committed_best is not None:
                del committed_best
            committed_best = torch.load(
                PILOT_BEST_CHECKPOINT_PATH,
                map_location='cpu',
                weights_only=False,
            )
        best_checkpoint_resume_checks = {
            'best_checkpoint_identity_matches': (
                committed_best.get('pilot_version') == PILOT_VERSION
                and committed_best.get('code_revision')
                == PILOT_CODE_REVISION
                and committed_best.get('replication_series_version')
                == REPLICATION_SERIES_VERSION
                and committed_best.get('development_fold')
                == REPLICATION_FOLD
                and committed_best.get('config_hash') == config_hash
                and int(committed_best.get('parameter_count', -1))
                == EXPECTED_V22_PARAMETERS
            ),
            'best_checkpoint_epoch_matches_history_argmax': (
                int(committed_best.get('epoch', -1)) == numeric_best_epoch
            ),
            'best_checkpoint_numeric_epoch_matches': (
                int(committed_best.get('numeric_best_epoch', -1))
                == numeric_best_epoch
            ),
            'best_checkpoint_numeric_score_matches': pilot_float_matches(
                committed_best.get('numeric_best_score', float('nan')),
                numeric_best_score,
            ),
            'best_checkpoint_summary_score_matches': pilot_float_matches(
                committed_best.get('validation_summary', {}).get(
                    'pilot_checkpoint_score', float('nan')
                ),
                numeric_best_score,
            ),
        }
        del committed_best
        if not all(best_checkpoint_resume_checks.values()):
            failed = [
                name for name, passed in best_checkpoint_resume_checks.items()
                if not passed
            ]
            raise RuntimeError(
                'Strict-best checkpoint resume audit failed: '
                + ', '.join(failed)
            )
        if resume_discarded_uncommitted_history_epochs:
            atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        resume_stop_already_triggered = bool(
            checkpoint_epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        )
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        if PILOT_BEST_CHECKPOINT_PATH.exists() or PILOT_HISTORY_PATH.exists():
            raise RuntimeError(
                'A strict-best checkpoint or committed history exists without '
                'a last checkpoint; refusing ambiguous recovery in the '
                'final-study directory.'
            )
        if not FRESH_START_FROM_INITIALIZATION:
            raise RuntimeError(
                'R5 fresh-start contract was unexpectedly disabled.'
            )
        if PARENT_CHECKPOINT_MIGRATION_ALLOWED:
            raise RuntimeError(
                'R5 parent-checkpoint migration was unexpectedly enabled.'
            )
        print(
            f'\nNo fold-{REPLICATION_FOLD} final-study checkpoint exists. '
            'Starting from deterministic model '
            'initialization; no parent checkpoint was read.'
        )

    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} STARTING'
    )
    print('=' * 80)
    print(f'Pilot version:                       {PILOT_VERSION}')
    print(
        'Fit / held-out subjects:             '
        f'{len(active_fit_df)} / {len(active_heldout_df)}'
    )
    print(f'Samples per epoch:                   {len(dataset)}')
    print(f'Parameter count:                     {EXPECTED_V22_PARAMETERS:,}')
    print(f'Resume epoch:                        {start_epoch}')
    print('Fresh-start lineage:                 True')
    print('Parent checkpoint migration allowed: False')
    print('Metrics printed:                     every epoch')
    print('Checkpoint selection:                strict numerical best')
    print(
        'Patience reset delta:                '
        f'{EARLY_STOPPING_MINIMUM_DELTA:.3f} (independent)'
    )
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Fold-0 architecture-development role: acknowledged')
    print(f'Completed replication folds:         {COMPLETED_REPLICATION_FOLDS}')
    if resume_discarded_uncommitted_history_epochs:
        print(
            'Recovered uncommitted history rows:   '
            f'{resume_discarded_uncommitted_history_epochs}'
        )
    if resume_recovered_strict_best_from_last:
        print('Recovered strict-best from last ckpt: True')

    stop_reason = (
        'early_stopping_resume_finalize'
        if resume_stop_already_triggered else 'maximum_epochs'
    )
    last_summary = None
    last_tier_rows = []
    last_branch_frame = pd.DataFrame()
    epoch_iterator = (
        range(0) if resume_stop_already_triggered
        else range(start_epoch, MAX_EPOCHS + 1)
    )
    for epoch in epoch_iterator:
        epoch_start = time.time()
        dataset.set_epoch(epoch)
        generator = torch.Generator()
        generator.manual_seed(PILOT_SEED + epoch * 97_409)
        loader = torch.utils.data.DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0,
            pin_memory=True, generator=generator,
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = defaultdict(float)
        processed = 0
        epoch_gradient_norms = []
        for batch_index, cpu_batch in enumerate(loader):
            with torch.autocast(
                device_type='cuda', dtype=torch.bfloat16, enabled=True
            ):
                loss, parts = pilot_compute_training_loss(
                    model, cpu_batch, device, epoch, batch_index
                )
                scaled_loss = loss / ACCUMULATION_STEPS
            scaled_loss.backward()
            processed += 1
            for name, value in parts.items():
                running[name] += float(value.detach().float().cpu())
            should_step = (
                (batch_index + 1) % ACCUMULATION_STEPS == 0
                or (batch_index + 1) == len(loader)
            )
            if should_step:
                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRADIENT_CLIP_NORM
                )
                if not bool(torch.isfinite(gradient_norm).item()):
                    raise RuntimeError(
                        f'Epoch {epoch}: non-finite gradient norm.'
                    )
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update_step += 1
                pilot_update_ema(ema_model, model, update_step)
                epoch_gradient_norms.append(float(gradient_norm.cpu()))

        summary, tier_rows = pilot_validate(ema_model, epoch, device)
        branch_frame = pd.DataFrame()
        if epoch == 1 or epoch % 5 == 0:
            branch_frame = pilot_branch_reliance(ema_model, epoch, device)
        current_score = float(summary['pilot_checkpoint_score'])
        if not math.isfinite(current_score):
            raise RuntimeError(
                f'Epoch {epoch}: composite validation score is non-finite.'
            )
        numeric_best_improved = current_score > numeric_best_score
        if numeric_best_improved:
            numeric_best_score = current_score
            numeric_best_epoch = epoch
            numeric_best_threshold = float(summary['threshold'])
            numeric_best_proposal_threshold = float(
                summary['proposal_threshold']
            )
        patience_meaningfully_improved = (
            current_score
            > patience_reference_score + EARLY_STOPPING_MINIMUM_DELTA
        )
        if patience_meaningfully_improved:
            patience_reference_score = current_score
            patience_reference_epoch = epoch
            validations_without_meaningful_improvement = 0
        elif epoch < MINIMUM_EPOCHS:
            
            
            
            validations_without_meaningful_improvement = 0
        else:
            validations_without_meaningful_improvement += 1
        elapsed_minutes = (time.time() - epoch_start) / 60.0
        history_row = {
            'epoch': epoch,
            **{f'train_{name}': value / max(processed, 1)
               for name, value in running.items()},
            'gradient_norm_mean': float(np.mean(epoch_gradient_norms)),
            'learning_rate_preserved': optimizer.param_groups[0]['lr'],
            'learning_rate_new': optimizer.param_groups[1]['lr'],
            'learning_rate_gate': optimizer.param_groups[2]['lr'],
            'validation_mean_dice': summary['positive_mean_dice'],
            'validation_median_dice': summary['positive_median_dice'],
            'validation_global_dice': summary['positive_global_dice'],
            'validation_lesion_f1': summary['lesion_f1'],
            'validation_lesion_recall': summary['lesion_recall'],
            'validation_component_recall': summary['component_recall'],
            'validation_small_component_recall': summary[
                'small_component_recall'
            ],
            'proposal_component_recall': summary[
                'proposal_component_recall'
            ],
            'proposal_global_dice': summary['proposal_global_dice'],
            'representation_failures': summary['representation_failures'],
            'proposal_failures': summary['proposal_failures'],
            'refinement_failures': summary['refinement_failures'],
            'local_threshold': summary['threshold'],
            'proposal_threshold': summary['proposal_threshold'],
            'pilot_checkpoint_score': current_score,
            **{
                f'validation_{lesion_bin}_mean_dice': float(
                    summary.get('lesion_bin_mean_dice', {}).get(
                        lesion_bin, float('nan')
                    )
                )
                for lesion_bin in [
                    'tiny', 'small', 'medium', 'large', 'very_large'
                ]
            },
            'presence_brier_score': summary['presence_brier_score'],
            'router_volume_mae_ml': summary['router_volume_mae_ml'],
            'empty_false_positive_volume_ml': summary[
                'empty_false_positive_volume_ml'
            ],
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_improved': numeric_best_improved,
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'patience_meaningfully_improved': (
                patience_meaningfully_improved
            ),
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'elapsed_minutes': elapsed_minutes,
        }
        history_rows = [
            row for row in history_rows if int(row['epoch']) != epoch
        ] + [history_row]
        history_rows = sorted(history_rows, key=lambda row: int(row['epoch']))
        atomic_write_csv(pd.DataFrame(history_rows), PILOT_HISTORY_PATH)
        checkpoint_record = {
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'config_hash': config_hash,
            'parameter_count': EXPECTED_V22_PARAMETERS,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_selection_minimum_delta': (
                CHECKPOINT_SELECTION_MINIMUM_DELTA
            ),
            'checkpoint_and_patience_state_are_independent': True,
            'resume_recovered_strict_best_from_last': (
                resume_recovered_strict_best_from_last
            ),
            'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
            'epoch': epoch, 'update_step': update_step,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'rng_state': pilot_capture_rng_state(),
            'numeric_best_score': numeric_best_score,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'validation_summary': summary,
        }
        pilot_atomic_torch_save(checkpoint_record, PILOT_LAST_CHECKPOINT_PATH)
        if numeric_best_improved:
            pilot_atomic_torch_save(checkpoint_record, PILOT_BEST_CHECKPOINT_PATH)
        atomic_write_json({
            'pilot_version': PILOT_VERSION,
            'code_revision': PILOT_CODE_REVISION,
            'replication_series_version': REPLICATION_SERIES_VERSION,
            'development_fold': REPLICATION_FOLD,
            'status': 'running', 'last_completed_epoch': epoch,
            'numeric_best_epoch': numeric_best_epoch,
            'numeric_best_score': numeric_best_score,
            'numeric_best_threshold': numeric_best_threshold,
            'numeric_best_proposal_threshold': (
                numeric_best_proposal_threshold
            ),
            'patience_reference_score': patience_reference_score,
            'patience_reference_epoch': patience_reference_epoch,
            'validations_without_meaningful_improvement': (
                validations_without_meaningful_improvement
            ),
            'checkpoint_selection_policy': 'strict_numerical_maximum',
            'checkpoint_and_patience_state_are_independent': True,
            'fresh_start_from_initialization': True,
            'parent_checkpoint_migration_allowed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'development_fivefold_replication_authorized': True,
            'early_stopping_triggered': bool(
                epoch >= MINIMUM_EPOCHS
                and validations_without_meaningful_improvement
                >= EARLY_STOPPING_PATIENCE
            ),
        }, PILOT_STATE_PATH)
        print(
            f'Epoch {epoch:03d}/{MAX_EPOCHS}: '
            f'loss={running["total_loss"]/max(processed,1):.4f}, '
            f'main={running["main_loss"]/max(processed,1):.4f}, '
            f'proposal={running["proposal_loss"]/max(processed,1):.4f}, '
            f'grad={np.mean(epoch_gradient_norms):.3f}, '
            f'time={elapsed_minutes:.1f} min'
        )
        print(
            f'  Validation: mean Dice={summary["positive_mean_dice"]:.4f}, '
            f'median={summary["positive_median_dice"]:.4f}, '
            f'global={summary["positive_global_dice"]:.4f}, '
            f'lesion F1/recall={summary["lesion_f1"]:.4f}/'
            f'{summary["lesion_recall"]:.4f}'
        )
        print(
            f'  Components: final all/small={summary["component_recall"]:.4f}/'
            f'{summary["small_component_recall"]:.4f}, '
            f'proposal recall/Dice={summary["proposal_component_recall"]:.4f}/'
            f'{summary["proposal_global_dice"]:.4f}, '
            f'failures proposal/refine/representation='
            f'{summary["proposal_failures"]}/{summary["refinement_failures"]}/'
            f'{summary["representation_failures"]}'
        )
        bin_metrics = summary.get('lesion_bin_mean_dice', {})
        print(
            '  Size-bin Dice: '
            + ', '.join(
                f'{name}={float(bin_metrics.get(name, float("nan"))):.4f}'
                for name in ['tiny', 'small', 'medium', 'large', 'very_large']
            )
        )
        print(
            f'  Calibration: local={summary["threshold"]:.2f}, '
            f'proposal={summary["proposal_threshold"]:.2f}, '
            f'score={current_score:.4f}, numeric best={numeric_best_score:.4f} '
            f'at epoch {numeric_best_epoch}'
        )
        print(
            f'  Stopping: reference={patience_reference_score:.4f} '
            f'at epoch {patience_reference_epoch}, patience='
            f'{validations_without_meaningful_improvement}/'
            f'{EARLY_STOPPING_PATIENCE}, '
            f'numeric_saved={numeric_best_improved}, '
            f'meaningful_reset={patience_meaningfully_improved}'
        )
        last_summary, last_tier_rows = summary, tier_rows
        if len(branch_frame):
            last_branch_frame = branch_frame
        if (
            epoch >= MINIMUM_EPOCHS
            and validations_without_meaningful_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            stop_reason = 'early_stopping'
            break

    if not PILOT_BEST_CHECKPOINT_PATH.exists():
        raise RuntimeError('Pilot ended without a best checkpoint.')
    if not PILOT_HISTORY_PATH.exists() or not PILOT_LAST_CHECKPOINT_PATH.exists():
        raise RuntimeError(
            'Final strict-best audit requires history and last checkpoint.'
        )
    committed_history_df = pd.read_csv(PILOT_HISTORY_PATH)
    committed_history_rows = committed_history_df.to_dict('records')
    reconstructed_final_state = pilot_recompute_selection_state(
        committed_history_rows
    )
    best_checkpoint = torch.load(
        PILOT_BEST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    last_checkpoint = torch.load(
        PILOT_LAST_CHECKPOINT_PATH,
        map_location='cpu',
        weights_only=False,
    )
    best_epoch = int(best_checkpoint['numeric_best_epoch'])
    best_score = float(best_checkpoint['numeric_best_score'])
    best_threshold = float(best_checkpoint['numeric_best_threshold'])
    best_proposal_threshold = float(
        best_checkpoint['numeric_best_proposal_threshold']
    )
    best_summary = dict(best_checkpoint['validation_summary'])
    strict_best_checks = {
        'history_is_nonempty': len(committed_history_df) > 0,
        'best_checkpoint_identity_matches_final_protocol': (
            best_checkpoint.get('pilot_version') == PILOT_VERSION
            and best_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and best_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and best_checkpoint.get('development_fold') == REPLICATION_FOLD
            and best_checkpoint.get('config_hash') == config_hash
            and int(best_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'last_checkpoint_identity_matches_final_protocol': (
            last_checkpoint.get('pilot_version') == PILOT_VERSION
            and last_checkpoint.get('code_revision') == PILOT_CODE_REVISION
            and last_checkpoint.get('replication_series_version')
            == REPLICATION_SERIES_VERSION
            and last_checkpoint.get('development_fold') == REPLICATION_FOLD
            and last_checkpoint.get('config_hash') == config_hash
            and int(last_checkpoint.get('parameter_count', -1))
            == EXPECTED_V22_PARAMETERS
        ),
        'history_argmax_epoch_matches_best_checkpoint': (
            reconstructed_final_state['numeric_best_epoch'] == best_epoch
        ),
        'history_argmax_score_matches_best_checkpoint': pilot_float_matches(
            reconstructed_final_state['numeric_best_score'], best_score
        ),
        'best_checkpoint_was_written_at_argmax_epoch': (
            int(best_checkpoint.get('epoch', -1)) == best_epoch
        ),
        'best_checkpoint_summary_score_matches_argmax': pilot_float_matches(
            best_summary.get('pilot_checkpoint_score', float('nan')),
            best_score,
        ),
        'best_checkpoint_local_threshold_matches_summary': pilot_float_matches(
            best_threshold, best_summary.get('threshold', float('nan'))
        ),
        'best_checkpoint_proposal_threshold_matches_summary': (
            pilot_float_matches(
                best_proposal_threshold,
                best_summary.get('proposal_threshold', float('nan')),
            )
        ),
        'best_checkpoint_policy_is_strict_numeric': (
            best_checkpoint.get('checkpoint_selection_policy')
            == 'strict_numerical_maximum'
            and float(
                best_checkpoint.get(
                    'checkpoint_selection_minimum_delta', float('nan')
                )
            ) == CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'best_checkpoint_patience_is_independent': (
            best_checkpoint.get(
                'checkpoint_and_patience_state_are_independent'
            ) is True
            and float(
                best_checkpoint.get('patience_minimum_delta', float('nan'))
            ) == EARLY_STOPPING_MINIMUM_DELTA
        ),
        'last_checkpoint_matches_history_end': (
            int(last_checkpoint.get('epoch', -1))
            == reconstructed_final_state['last_epoch']
        ),
        'last_checkpoint_numeric_best_epoch_matches_history': (
            int(last_checkpoint.get('numeric_best_epoch', -1)) == best_epoch
        ),
        'last_checkpoint_numeric_best_score_matches_history': (
            pilot_float_matches(
                last_checkpoint.get('numeric_best_score', float('nan')),
                best_score,
            )
        ),
        'last_checkpoint_patience_reference_matches_history': (
            int(last_checkpoint.get('patience_reference_epoch', -1))
            == reconstructed_final_state['patience_reference_epoch']
            and pilot_float_matches(
                last_checkpoint.get(
                    'patience_reference_score', float('nan')
                ),
                reconstructed_final_state['patience_reference_score'],
            )
        ),
        'last_checkpoint_patience_counter_matches_history': (
            int(last_checkpoint.get(
                'validations_without_meaningful_improvement', -1
            )) == reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'history_does_not_exceed_maximum_epochs': (
            reconstructed_final_state['last_epoch'] <= MAX_EPOCHS
        ),
    }
    if not all(strict_best_checks.values()):
        failed = [
            name for name, passed in strict_best_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final strict-best checkpoint audit failed: ' + ', '.join(failed)
        )
    ema_model.load_state_dict(best_checkpoint['ema_model_state'])
    del last_checkpoint
    del best_checkpoint
    history_max_mean_row = committed_history_df.loc[
        committed_history_df['validation_mean_dice'].idxmax()
    ]
    history_max_global_row = committed_history_df.loc[
        committed_history_df['validation_global_dice'].idxmax()
    ]
    best_tier_path = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
        / 'component_tier_metrics.csv'
    )
    best_validation_root = (
        PILOT_VALIDATION_ROOT / f'epoch_{best_epoch:03d}'
    )
    best_local_threshold_case_path = (
        best_validation_root / 'local_threshold_case_metrics.csv'
    )
    best_proposal_threshold_case_path = (
        best_validation_root / 'proposal_threshold_case_metrics.csv'
    )
    for required_best_artifact in [
        best_tier_path,
        best_local_threshold_case_path,
        best_proposal_threshold_case_path,
    ]:
        if not required_best_artifact.exists():
            raise RuntimeError(
                'Best-epoch OOF artifact is missing: '
                f'{required_best_artifact}'
            )
    best_local_threshold_case_df = pd.read_csv(
        best_local_threshold_case_path
    )
    best_proposal_threshold_case_df = pd.read_csv(
        best_proposal_threshold_case_path
    )
    observed_local_thresholds = np.sort(
        best_local_threshold_case_df['threshold'].unique()
    )
    observed_proposal_thresholds = np.sort(
        best_proposal_threshold_case_df['threshold'].unique()
    )
    best_oof_checks = {
        'local_oof_rows_complete': (
            len(best_local_threshold_case_df)
            == len(active_heldout_df) * len(LOCAL_THRESHOLDS)
        ),
        'proposal_oof_rows_complete': (
            len(best_proposal_threshold_case_df)
            == len(active_heldout_df) * len(PROPOSAL_THRESHOLDS)
        ),
        'local_oof_subjects_exactly_heldout': (
            set(best_local_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'proposal_oof_subjects_exactly_heldout': (
            set(best_proposal_threshold_case_df['subject_id'].astype(str))
            == heldout_subject_ids
        ),
        'local_oof_threshold_grid_exact': (
            observed_local_thresholds.shape == LOCAL_THRESHOLDS.shape
            and np.allclose(
                observed_local_thresholds, LOCAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'proposal_oof_threshold_grid_exact': (
            observed_proposal_thresholds.shape == PROPOSAL_THRESHOLDS.shape
            and np.allclose(
                observed_proposal_thresholds, PROPOSAL_THRESHOLDS,
                rtol=0.0, atol=1e-8,
            )
        ),
        'oof_records_are_active_fold_only': (
            set(
                best_local_threshold_case_df['development_fold'].astype(int)
            ) == {REPLICATION_FOLD}
            and set(
                best_proposal_threshold_case_df[
                    'development_fold'
                ].astype(int)
            ) == {REPLICATION_FOLD}
        ),
        'oof_records_are_best_epoch_only': (
            set(best_local_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
            and set(best_proposal_threshold_case_df['epoch'].astype(int))
            == {best_epoch}
        ),
    }
    if not all(best_oof_checks.values()):
        failed = [
            name for name, passed in best_oof_checks.items() if not passed
        ]
        raise RuntimeError(
            'Best-epoch OOF sufficient-statistics audit failed: '
            + ', '.join(failed)
        )
    best_tier_rows = pd.read_csv(best_tier_path).to_dict('records')
    best_branch_frame = pilot_branch_reliance(ema_model, best_epoch, device)
    best_artifact_paths = {
        'validation_summary': (
            best_validation_root / 'validation_summary.json'
        ),
        'case_metrics': best_validation_root / 'case_metrics.csv',
        'component_metrics': best_validation_root / 'component_metrics.csv',
        'component_tier_metrics': best_tier_path,
        'local_threshold_summary': (
            best_validation_root / 'local_threshold_summary.csv'
        ),
        'local_threshold_case_metrics': best_local_threshold_case_path,
        'proposal_threshold_summary': (
            best_validation_root / 'proposal_threshold_summary.csv'
        ),
        'proposal_threshold_case_metrics': (
            best_proposal_threshold_case_path
        ),
        'branch_reliance': (
            PILOT_DIAGNOSTIC_ROOT
            / f'epoch_{best_epoch:03d}_branch_reliance.csv'
        ),
        'branch_reliance_history': PILOT_BRANCH_HISTORY_PATH,
        'fit_target_access_audit': PILOT_TARGET_AUDIT_PATH,
        'fit_component_audit': PILOT_COMPONENT_AUDIT_PATH,
        'epoch1_schedule_audit': PILOT_SCHEDULE_AUDIT_PATH,
        'epoch1_negative_skip_audit': PILOT_NEGATIVE_SKIP_AUDIT_PATH,
    }
    missing_best_artifacts = [
        str(path) for path in best_artifact_paths.values()
        if not Path(path).exists()
    ]
    if missing_best_artifacts:
        raise RuntimeError(
            'Final fold artifact bundle is incomplete:\n'
            + '\n'.join(missing_best_artifacts)
        )
    best_artifact_manifest = {
        name: {
            'path': str(path),
            'sha256': file_sha256(path),
        }
        for name, path in best_artifact_paths.items()
    }
    findings, branch_summary, recommendation = pilot_diagnostic_interpretation(
        best_summary, best_tier_rows, best_branch_frame
    )
    r5_source_checkpoint_hash_after = file_sha256(
        R5_SOURCE_BEST_CHECKPOINT_PATH
    )
    if r5_source_checkpoint_hash_after != r5_source_checkpoint_hash_before:
        raise RuntimeError(
            'The selected R5 source checkpoint changed during fold '
            'replication. Refusing to commit a result gate.'
        )
    final_stop_checks = {
        'early_stop_state_is_consistent': (
            not stop_reason.startswith('early_stopping')
            or (
                reconstructed_final_state['last_epoch'] >= MINIMUM_EPOCHS
                and reconstructed_final_state[
                    'validations_without_meaningful_improvement'
                ] >= EARLY_STOPPING_PATIENCE
            )
        ),
        'maximum_epoch_state_is_consistent': (
            stop_reason != 'maximum_epochs'
            or reconstructed_final_state['last_epoch'] == MAX_EPOCHS
        ),
    }
    if not all(final_stop_checks.values()):
        failed = [
            name for name, passed in final_stop_checks.items() if not passed
        ]
        raise RuntimeError(
            'Final stopping-state audit failed: ' + ', '.join(failed)
        )
    result_gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'pilot_version': PILOT_VERSION,
        'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'parameter_count': EXPECTED_V22_PARAMETERS,
        'pilot_complete': True,
        'replication_complete': True,
        'final_fold_complete': True,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'stop_reason': stop_reason,
        'last_completed_epoch': int(
            reconstructed_final_state['last_epoch']
        ),
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_threshold': best_threshold,
        'best_proposal_threshold': best_proposal_threshold,
        'best_validation_summary': best_summary,
        'strict_numeric_best_verified': True,
        'strict_best_checks': strict_best_checks,
        'checkpoint_selection_policy': 'strict_numerical_maximum',
        'checkpoint_selection_minimum_delta': (
            CHECKPOINT_SELECTION_MINIMUM_DELTA
        ),
        'checkpoint_and_patience_state_are_independent': True,
        'patience_minimum_delta': EARLY_STOPPING_MINIMUM_DELTA,
        'patience_reference_score': float(
            reconstructed_final_state['patience_reference_score']
        ),
        'patience_reference_epoch': int(
            reconstructed_final_state['patience_reference_epoch']
        ),
        'final_validations_without_meaningful_improvement': int(
            reconstructed_final_state[
                'validations_without_meaningful_improvement'
            ]
        ),
        'final_stop_checks': final_stop_checks,
        'history_max_mean_dice': float(
            history_max_mean_row['validation_mean_dice']
        ),
        'history_max_mean_dice_epoch': int(history_max_mean_row['epoch']),
        'history_max_global_dice': float(
            history_max_global_row['validation_global_dice']
        ),
        'history_max_global_dice_epoch': int(history_max_global_row['epoch']),
        'history_max_mean_and_global_are_secondary_diagnostics_only': True,
        'diagnostic_findings': findings,
        'branch_target_probability_deltas': branch_summary,
        'diagnostic_recommendation': recommendation,
        'recommended_next_action': (
            'Build the pooled out-of-fold aggregation and frozen-threshold '
            'block.'
            if REPLICATION_FOLD == AUTHORIZED_REPLICATION_FOLDS[-1]
            else
            'Rerun this unchanged block to advance to the next authorized '
            'development fold.'
        ),
        'fit_subjects': len(active_fit_df),
        'heldout_subjects': len(active_heldout_df),
        'fit_subject_hash': pilot_subject_hash(active_fit_df['subject_id']),
        'heldout_subject_hash': pilot_subject_hash(
            active_heldout_df['subject_id']
        ),
        'pilot_config_path': str(PILOT_CONFIG_PATH),
        'pilot_config_sha256': file_sha256(PILOT_CONFIG_PATH),
        'authorization_gate_path': str(AUTHORIZATION_GATE_PATH),
        'authorization_gate_sha256': file_sha256(AUTHORIZATION_GATE_PATH),
        'r5_source_checkpoint_sha256': r5_source_checkpoint_hash_before,
        'r5_source_checkpoint_unchanged': True,
        'r5_source_checkpoint_loaded_for_initialization': False,
        'resume_discarded_uncommitted_history_epochs': (
            resume_discarded_uncommitted_history_epochs
        ),
        'resume_recovered_strict_best_from_last': (
            resume_recovered_strict_best_from_last
        ),
        'best_checkpoint_path': str(PILOT_BEST_CHECKPOINT_PATH),
        'best_checkpoint_sha256': file_sha256(PILOT_BEST_CHECKPOINT_PATH),
        'last_checkpoint_path': str(PILOT_LAST_CHECKPOINT_PATH),
        'last_checkpoint_sha256': file_sha256(PILOT_LAST_CHECKPOINT_PATH),
        'history_path': str(PILOT_HISTORY_PATH),
        'history_sha256': file_sha256(PILOT_HISTORY_PATH),
        'best_epoch_artifact_manifest': best_artifact_manifest,
        'best_local_threshold_case_metrics_path': str(
            best_local_threshold_case_path
        ),
        'best_local_threshold_case_metrics_sha256': file_sha256(
            best_local_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_path': str(
            best_proposal_threshold_case_path
        ),
        'best_proposal_threshold_case_metrics_sha256': file_sha256(
            best_proposal_threshold_case_path
        ),
        'pooled_oof_threshold_calibration_sufficient_statistics_written': True,
        'fold_selected_threshold_is_not_final_deployment_threshold': True,
        'best_oof_sufficient_statistics_checks': best_oof_checks,
        'target_access_checks': pilot_target_access_checks,
        'target_arrays_read_splits': [
            f'train_fold{REPLICATION_FOLD}_fit',
            f'train_fold{REPLICATION_FOLD}_heldout',
        ],
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'active_fold_training_authorized': True,
        'development_fivefold_replication_authorized': True,
        'final_protocol_role': FINAL_PROTOCOL_ROLE,
        'final_protocol_retrains_fold0': FINAL_PROTOCOL_RETRAINS_FOLD0,
        'fold0_architecture_development_bias_acknowledged': True,
        'architecture_changed_from_r5': False,
        'optimization_loss_sampling_changed_from_r5': False,
        'checkpoint_selection_semantics_changed_from_r5': True,
        'manual_review_required_before_next_fold': False,
    }
    atomic_write_json(result_gate, PILOT_RESULT_GATE_PATH)
    atomic_write_json({
        'pilot_version': PILOT_VERSION, 'code_revision': PILOT_CODE_REVISION,
        'replication_series_version': REPLICATION_SERIES_VERSION,
        'development_fold': REPLICATION_FOLD,
        'status': 'complete', 'best_epoch': best_epoch,
        'best_score': result_gate['best_score'],
        'stop_reason': stop_reason,
        'fresh_start_from_initialization': True,
        'parent_checkpoint_migration_allowed': False,
        'development_fivefold_replication_authorized': True,
        'strict_numeric_best_verified': True,
        'checkpoint_and_patience_state_are_independent': True,
    }, PILOT_STATE_PATH)
    series_status = write_final_series_status()
    print('\n' + '=' * 80)
    print(
        f'TRIAL8 V2.2 R5 FINAL STUDY FOLD-{REPLICATION_FOLD} COMPLETE'
    )
    print('=' * 80)
    print(f'Best epoch:                         {best_epoch}')
    print(f'Best pilot score:                   {result_gate["best_score"]:.4f}')
    print(f'Best mean Dice:                     {best_summary["positive_mean_dice"]:.4f}')
    print(f'Best global Dice:                   {best_summary["positive_global_dice"]:.4f}')
    print(f'Best lesion F1:                     {best_summary["lesion_f1"]:.4f}')
    print('Strict numerical best verified:      True')
    print(
        'Patience reference / epoch:          '
        f'{result_gate["patience_reference_score"]:.4f} / '
        f'{result_gate["patience_reference_epoch"]}'
    )
    print(f'Final component recall:             {best_summary["component_recall"]:.4f}')
    print(f'Proposal component recall:          {best_summary["proposal_component_recall"]:.4f}')
    print(f'Diagnostic interpretation:          {recommendation}')
    print('Locked validation targets read:      False')
    print('Test targets read:                   False')
    print('Final protocol folds 0-4:             True')
    print('Architecture/optimization changed:   False')
    print('Checkpoint semantics corrected:      True')
    print(f'Best checkpoint:                    {PILOT_BEST_CHECKPOINT_PATH}')
    print(f'Result gate:                        {PILOT_RESULT_GATE_PATH}')
    print(f'Series status:                      {FINAL_SERIES_STATUS_PATH}')
    print(
        'Completed final-protocol folds:      '
        f'{tuple(series_status["completed_folds"])}'
    )















LOCKED_EVALUATION_VERSION = 'v3_final_locked_val_test_metric_reconciliation'
LOCKED_EVALUATION_CODE_REVISION = (
    'r3_target_derived_metric_eligibility_reuse_frozen_r2_predictions'
)
SOURCE_PREDICTION_VERSION = 'v2_final_locked_val_test_fivefold_ensemble_once'
SOURCE_PREDICTION_CODE_REVISION = (
    'r2_locked_input_contract_training_lock_semantics'
)
OOF_FREEZE_VERSION = 'v1_final_fivefold_oof_threshold_freeze'
EXPECTED_OOF_AGGREGATION_VERSION = OOF_FREEZE_VERSION
EXPECTED_OOF_CODE_REVISION = (
    'r1_exact_sufficient_statistics_locked_policy'
)
EXPECTED_ENSEMBLE_FOLDS = (0, 1, 2, 3, 4)
EXPECTED_LOCKED_SPLITS = {'val': 23, 'test': 23}
EXPECTED_FROZEN_LOCAL_THRESHOLD = 0.40
EXPECTED_FROZEN_PROPOSAL_THRESHOLD = 0.35
LOCKED_BOOTSTRAP_REPLICATES = 10_000
LOCKED_BOOTSTRAP_SEED = 20261012

OOF_FREEZE_ROOT = (
    PROJECT_ROOT / 'preprocessing' / R2_VERSION / 'evaluation_preflight'
    / MODEL_VERSION / OOF_FREEZE_VERSION
)
OOF_DEPLOYMENT_POLICY_PATH = OOF_FREEZE_ROOT / 'frozen_deployment_policy.json'
OOF_AGGREGATION_GATE_PATH = OOF_FREEZE_ROOT / 'oof_threshold_freeze_gate.json'

LOCKED_OUTPUT_ROOT = (
    PROJECT_ROOT / 'evaluation' / MODEL_VERSION / LOCKED_EVALUATION_VERSION
)
SOURCE_PREDICTION_ROOT = (
    PROJECT_ROOT / 'evaluation' / MODEL_VERSION / SOURCE_PREDICTION_VERSION
)
LOCKED_PREDICTION_ROOT = SOURCE_PREDICTION_ROOT / 'target_blind_predictions'
LOCKED_PREDICTION_INDEX_PATH = SOURCE_PREDICTION_ROOT / 'prediction_index.json'
LOCKED_PREDICTION_GATE_PATH = (
    SOURCE_PREDICTION_ROOT / 'target_blind_prediction_gate.json'
)
SOURCE_FLAWED_FINAL_GATE_PATH = SOURCE_PREDICTION_ROOT / 'locked_evaluation_gate.json'
LOCKED_SOURCE_MANIFEST_PATH = LOCKED_OUTPUT_ROOT / 'source_artifact_manifest.csv'
LOCKED_PREDICTION_AUDIT_PATH = LOCKED_OUTPUT_ROOT / 'prediction_audit.csv'
LOCKED_TARGET_POLICY_AUDIT_PATH = (
    LOCKED_OUTPUT_ROOT / 'target_derived_policy_audit.csv'
)
LOCKED_CASE_METRICS_PATH = LOCKED_OUTPUT_ROOT / 'locked_case_metrics.csv'
LOCKED_COMPONENT_METRICS_PATH = (
    LOCKED_OUTPUT_ROOT / 'locked_component_metrics.csv'
)
LOCKED_SPLIT_SUMMARY_PATH = LOCKED_OUTPUT_ROOT / 'locked_split_summary.csv'
LOCKED_BIN_METRICS_PATH = LOCKED_OUTPUT_ROOT / 'locked_lesion_bin_metrics.csv'
LOCKED_MEMBER_METRICS_PATH = (
    LOCKED_OUTPUT_ROOT / 'ensemble_member_metrics.csv'
)
LOCKED_DIVERSITY_PATH = LOCKED_OUTPUT_ROOT / 'ensemble_diversity_metrics.csv'
LOCKED_BOOTSTRAP_PATH = LOCKED_OUTPUT_ROOT / 'locked_bootstrap_intervals.csv'
LOCKED_GENERALIZATION_PATH = (
    LOCKED_OUTPUT_ROOT / 'oof_to_locked_generalization_summary.csv'
)
LOCKED_REPORT_PATH = LOCKED_OUTPUT_ROOT / 'locked_evaluation_report.json'
LOCKED_FINAL_GATE_PATH = LOCKED_OUTPUT_ROOT / 'locked_evaluation_gate.json'


def locked_checks_or_raise(checks, context):
    failed = [name for name, passed in checks.items() if not bool(passed)]
    if failed:
        raise RuntimeError(context + ': ' + ', '.join(failed))


def locked_canonical_hash(record):
    payload = json.dumps(
        to_builtin(record), sort_keys=True, separators=(',', ':'),
        allow_nan=False,
    ).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def locked_source_record(path, role, development_fold=None):
    path = Path(path)
    stat = path.stat()
    return {
        'role': str(role),
        'development_fold': (
            int(development_fold) if development_fold is not None else -1
        ),
        'path': str(path),
        'sha256': file_sha256(path),
        'size_bytes': int(stat.st_size),
        'mtime_ns': int(stat.st_mtime_ns),
    }


def locked_sources_unchanged(records):
    checks = {}
    for index, record in enumerate(records):
        path = Path(record['path'])
        stat = path.stat() if path.exists() else None
        checks[f'source_{index:03d}_{record["role"]}_unchanged'] = bool(
            stat is not None
            and int(stat.st_size) == int(record['size_bytes'])
            and int(stat.st_mtime_ns) == int(record['mtime_ns'])
            and file_sha256(path) == record['sha256']
        )
    return checks


def locked_manifest_input_contract_checks(locked_rows):
    
    required_columns = {
        'subject_id', 'dev_split', 'vascufate_v2_npz_path',
        'trial8_npz_sha256', 'trial8_atlas_enabled',
        'trial8_atlas_vascular30_path', 'trial8_graph_sidecar_path',
        'trial8_graph_sidecar_sha256', 'trial8_clinical_sidecar_path',
        'trial8_clinical_sidecar_sha256',
        'trial8_v2_multimodal_loader_preflight_passed',
        'trial8_model_training_authorized',
    }
    missing = sorted(required_columns - set(locked_rows.columns))
    if missing:
        raise RuntimeError(
            'Locked manifest is missing required inference columns: '
            + ', '.join(missing)
        )

    path_columns = [
        'vascufate_v2_npz_path', 'trial8_graph_sidecar_path',
        'trial8_clinical_sidecar_path',
    ]
    paths_declared = locked_rows[path_columns].apply(
        lambda column: column.map(
            lambda value: isinstance(value, str) and bool(value.strip())
        )
    )
    paths_exist = locked_rows[path_columns].apply(
        lambda column: column.map(
            lambda value: isinstance(value, str) and Path(value).is_file()
        )
    )
    hash_columns = [
        'trial8_npz_sha256', 'trial8_graph_sidecar_sha256',
        'trial8_clinical_sidecar_sha256',
    ]
    hashes_declared = locked_rows[hash_columns].apply(
        lambda column: column.map(
            lambda value: (
                isinstance(value, str)
                and len(value.strip()) == 64
                and all(character in '0123456789abcdef'
                        for character in value.strip().lower())
            )
        )
    )
    atlas_paths_valid = []
    for row in locked_rows.to_dict('records'):
        if as_bool(row['trial8_atlas_enabled']):
            atlas_path = str(row['trial8_atlas_vascular30_path']).strip()
            atlas_paths_valid.append(bool(atlas_path) and Path(atlas_path).is_file())
        else:
            atlas_paths_valid.append(True)

    loader_passed = bool(
        locked_rows['trial8_v2_multimodal_loader_preflight_passed']
        .map(as_bool).all()
    )
    training_locked = not bool(
        locked_rows['trial8_model_training_authorized'].map(as_bool).any()
    )
    inputs_verified = bool(
        loader_passed
        and training_locked
        and paths_declared.to_numpy(dtype=bool).all()
        and paths_exist.to_numpy(dtype=bool).all()
        and hashes_declared.to_numpy(dtype=bool).all()
        and all(atlas_paths_valid)
    )
    return {
        'all_loader_preflights_passed': loader_passed,
        'training_remains_locked_for_all': training_locked,
        'all_model_input_paths_declared': bool(
            paths_declared.to_numpy(dtype=bool).all()
        ),
        'all_model_input_files_exist': bool(
            paths_exist.to_numpy(dtype=bool).all()
        ),
        'all_model_input_hashes_declared': bool(
            hashes_declared.to_numpy(dtype=bool).all()
        ),
        'all_enabled_atlas_paths_exist': bool(all(atlas_paths_valid)),
        'all_target_free_model_inputs_verified': inputs_verified,
    }


def locked_pack_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    return np.packbits(mask.reshape(-1), bitorder='little')


def locked_unpack_mask(bits, shape):
    count = int(np.prod(shape))
    values = np.unpackbits(
        np.asarray(bits, dtype=np.uint8), bitorder='little'
    )[:count]
    return values.reshape(tuple(int(v) for v in shape)).astype(bool)


def locked_atomic_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'wb') as handle:
        np.savez_compressed(handle, **arrays)
    os.replace(temporary, path)


def locked_prediction_path(subject_id):
    return LOCKED_PREDICTION_ROOT / f'{subject_id}_fivefold_ensemble_masks.npz'


def locked_load_target_free_case(row):
    
    subject_id = str(row['subject_id'])
    npz_path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(npz_path, allow_pickle=False) as data:
        image = np.asarray(data['image'], dtype=np.float32)
        channels = decode_strings(data['input_channels'])
        cow = np.asarray(data['cow_soft_union'], dtype=np.float32)
        lvo = np.asarray(data['lvo_proximity'], dtype=np.float32)
        lvo_present = bool(np.asarray(data['lvo_present']).reshape(-1)[0])
    if channels != EXPECTED_INPUT_CHANNELS:
        raise RuntimeError(f'{subject_id}: channel schema changed.')
    native_shape = tuple(int(v) for v in image.shape[1:])
    atlas_enabled = as_bool(row['trial8_atlas_enabled'])
    if atlas_enabled:
        region = np.asanyarray(
            nib.load(str(row['trial8_atlas_vascular30_path'])).dataobj
        )
        if region.ndim == 4:
            region = region[..., 0]
        region = np.rint(region).astype(np.int64)
    else:
        region = np.zeros(native_shape, dtype=np.int64)
    image_canvas, pad_start = symmetric_pad(
        image, GLOBAL_CANVAS_SHAPE, (1, 2, 3)
    )
    region_canvas, region_start = symmetric_pad(
        region, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    cow_canvas, cow_start = symmetric_pad(
        cow, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    lvo_canvas, lvo_start = symmetric_pad(
        lvo, GLOBAL_CANVAS_SHAPE, (0, 1, 2)
    )
    if not (pad_start == region_start == cow_start == lvo_start):
        raise RuntimeError(f'{subject_id}: inference padding starts disagree.')
    center_origin = tuple(
        (GLOBAL_CANVAS_SHAPE[i] - LOCAL_PATCH_SIZE[i]) // 2
        for i in range(3)
    )
    base_sample, _ = load_v22_subject(
        row, pilot_schema, multimodal_gate, local_origin=center_origin
    )
    return {
        'subject_id': subject_id,
        'row': row,
        'image_canvas': image_canvas,
        'region_canvas': region_canvas,
        'cow_canvas': cow_canvas,
        'lvo_canvas': lvo_canvas,
        'lvo_present': lvo_present,
        'native_shape': native_shape,
        'pad_start': tuple(int(v) for v in pad_start),
        'base_sample': base_sample,
    }


@torch.no_grad()
def locked_infer_case(model, row, device):
    case = locked_load_target_free_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(model, base_batch)
    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[i], LOCAL_PATCH_SIZE[i])
        for i in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][axis] + native_start[axis])
                    for axis in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = pilot_local_from_cache(model, batch, cache)
                patch_probability = torch.sigmoid(
                    output['logits'].float()
                )[0, 0].cpu().numpy()
                destination = tuple(
                    slice(
                        native_start[axis],
                        min(
                            native_start[axis] + LOCAL_PATCH_SIZE[axis],
                            native_shape[axis],
                        ),
                    )
                    for axis in range(3)
                )
                copied_shape = tuple(
                    item.stop - item.start for item in destination
                )
                source = tuple(
                    slice(0, copied_shape[axis]) for axis in range(3)
                )
                probability_sum[destination] += (
                    patch_probability[source] * PILOT_WINDOW_WEIGHT[source]
                )
                weight_sum[destination] += PILOT_WINDOW_WEIGHT[source]
    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete sliding-window coverage."
        )
    probability = probability_sum / weight_sum
    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(
            case['pad_start'][axis],
            case['pad_start'][axis] + native_shape[axis],
        )
        for axis in range(3)
    )
    proposal = proposal_canvas[native_slices].astype(np.float32)
    presence = float(torch.sigmoid(
        cache['routed']['lesion_presence_logit'].float()
    )[0].cpu())
    routed_volume_ml = float(torch.expm1(
        cache['routed']['log_volume_prediction'].float()
    )[0].clamp_min(0).cpu())
    del base_batch, cache
    return {
        'subject_id': case['subject_id'],
        'probability': probability.astype(np.float32),
        'proposal_probability': proposal,
        'presence_probability': presence,
        'routed_volume_ml': routed_volume_ml,
    }


def locked_validate_prediction_archive(path, expected_contract_hash, subject_id):
    path = Path(path)
    if not path.is_file():
        return None
    with np.load(path, allow_pickle=False) as data:
        required = {
            'subject_id', 'dev_split', 'native_shape', 'contract_sha256',
            'fold_ids', 'fold_local_masks_packed',
            'ensemble_local_mask_packed', 'ensemble_proposal_mask_packed',
            'fold_predicted_voxels', 'fold_presence_probabilities',
            'fold_routed_volume_ml', 'ensemble_local_probability_sha256',
            'ensemble_proposal_probability_sha256',
        }
        if not required.issubset(data.files):
            raise RuntimeError(f'{subject_id}: prediction archive is incomplete.')
        observed_subject = scalar_string(data['subject_id'])
        observed_contract = scalar_string(data['contract_sha256'])
        shape = tuple(int(v) for v in np.asarray(data['native_shape']).tolist())
        fold_ids = tuple(int(v) for v in np.asarray(data['fold_ids']).tolist())
        fold_bits = np.asarray(data['fold_local_masks_packed'], dtype=np.uint8)
        local_bits = np.asarray(
            data['ensemble_local_mask_packed'], dtype=np.uint8
        )
        proposal_bits = np.asarray(
            data['ensemble_proposal_mask_packed'], dtype=np.uint8
        )
    expected_bytes = (int(np.prod(shape)) + 7) // 8
    checks = {
        'subject_matches': observed_subject == subject_id,
        'contract_matches': observed_contract == expected_contract_hash,
        'shape_is_three_dimensional': len(shape) == 3,
        'fold_ids_exact': fold_ids == EXPECTED_ENSEMBLE_FOLDS,
        'fold_mask_shape_exact': fold_bits.shape == (5, expected_bytes),
        'ensemble_local_mask_size_exact': local_bits.size == expected_bytes,
        'ensemble_proposal_mask_size_exact': proposal_bits.size == expected_bytes,
    }
    locked_checks_or_raise(
        checks, f'{subject_id}: existing target-blind prediction rejected'
    )
    return {
        'subject_id': subject_id,
        'path': str(path),
        'sha256': file_sha256(path),
        'native_shape': list(shape),
    }


def locked_pairwise_dice(masks):
    values = []
    for left in range(len(masks)):
        for right in range(left + 1, len(masks)):
            denominator = int(masks[left].sum() + masks[right].sum())
            intersection = int((masks[left] & masks[right]).sum())
            values.append(
                2.0 * intersection / denominator if denominator else 1.0
            )
    return values


def locked_prediction_manifest_row(
    subject_id, split, archive_path, shape, fold_masks,
    ensemble_local, ensemble_proposal, local_probability,
    proposal_probability, fold_presence, fold_routed_volume,
):
    pairwise = locked_pairwise_dice(fold_masks)
    vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
    union_voxels = int((vote_count > 0).sum())
    disagreement_voxels = int(((vote_count > 0) & (vote_count < 5)).sum())
    return {
        'subject_id': subject_id,
        'dev_split': split,
        'prediction_path': str(archive_path),
        'prediction_sha256': file_sha256(archive_path),
        'native_shape': list(shape),
        'ensemble_local_voxels': int(ensemble_local.sum()),
        'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
        'fold_local_predicted_voxels': [int(mask.sum()) for mask in fold_masks],
        'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
        'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
        'disagreement_union_fraction': float(
            disagreement_voxels / max(union_voxels, 1)
        ),
        'ensemble_local_probability_mean': float(local_probability.mean()),
        'ensemble_local_probability_maximum': float(local_probability.max()),
        'ensemble_proposal_probability_mean': float(
            proposal_probability.mean()
        ),
        'ensemble_proposal_probability_maximum': float(
            proposal_probability.max()
        ),
        'mean_presence_probability': float(np.mean(fold_presence)),
        'mean_routed_volume_ml': float(np.mean(fold_routed_volume)),
    }


def locked_load_target(row, expected_shape):
    subject_id = str(row['subject_id'])
    path = Path(str(row['vascufate_v2_npz_path']))
    with np.load(path, allow_pickle=False) as data:
        if not {'target_binary', 'target_soft', 'target_spacing'}.issubset(
            data.files
        ):
            raise RuntimeError(f'{subject_id}: locked target arrays unavailable.')
        target_raw = np.asarray(data['target_binary'])
        soft = np.asarray(data['target_soft'], dtype=np.float32)
        spacing = np.asarray(data['target_spacing'], dtype=np.float32)
    target = target_raw.astype(np.uint8, copy=False) > 0
    checks = {
        'target_shape_matches_prediction': tuple(target.shape) == tuple(expected_shape),
        'soft_shape_matches_prediction': tuple(soft.shape) == tuple(expected_shape),
        'binary_target_is_finite': bool(np.isfinite(target_raw).all()),
        'binary_target_contains_only_zero_or_one': bool(
            np.isin(target_raw, [0, 1]).all()
        ),
        'spacing_has_three_values': spacing.shape == (3,),
        'spacing_is_finite_positive': bool(
            np.isfinite(spacing).all() and (spacing > 0).all()
        ),
        'soft_is_finite': bool(np.isfinite(soft).all()),
        'soft_is_nonnegative': bool((soft >= 0).all()),
    }
    locked_checks_or_raise(checks, f'{subject_id}: locked target integrity failed')
    return target, soft, spacing


def locked_derive_target_policy(target, soft, spacing):
    
    target = np.asarray(target, dtype=bool)
    soft = np.asarray(soft, dtype=np.float64)
    spacing = np.asarray(spacing, dtype=np.float64).reshape(-1)
    checks = {
        'target_and_soft_shapes_match': target.shape == soft.shape,
        'spacing_has_three_values': spacing.shape == (3,),
        'spacing_is_finite_positive': bool(
            np.isfinite(spacing).all() and (spacing > 0).all()
        ),
        'soft_is_finite_nonnegative': bool(
            np.isfinite(soft).all() and (soft >= 0).all()
        ),
    }
    locked_checks_or_raise(checks, 'Target-derived policy inputs are invalid')
    hard_voxels = int(target.sum())
    soft_support_voxels = int((soft > 0).sum())
    voxel_volume_ml = float(np.prod(spacing) / 1000.0)
    hard_volume_ml = float(hard_voxels * voxel_volume_ml)
    soft_volume_ml = float(soft.sum(dtype=np.float64) * voxel_volume_ml)
    if hard_voxels > 0:
        supervision_mode = 'hard_and_soft'
        hard_metric_eligible = True
        effective_volume_ml = hard_volume_ml
    elif soft_support_voxels > 0:
        supervision_mode = 'soft_only_subvoxel'
        hard_metric_eligible = False
        effective_volume_ml = soft_volume_ml
    else:
        supervision_mode = 'true_empty'
        hard_metric_eligible = True
        effective_volume_ml = 0.0
    if supervision_mode == 'true_empty':
        lesion_volume_bin = 'empty'
    elif effective_volume_ml < 1.0:
        lesion_volume_bin = 'tiny'
    elif effective_volume_ml < 5.0:
        lesion_volume_bin = 'small'
    elif effective_volume_ml < 25.0:
        lesion_volume_bin = 'medium'
    elif effective_volume_ml < 75.0:
        lesion_volume_bin = 'large'
    else:
        lesion_volume_bin = 'very_large'
    return {
        'supervision_mode': supervision_mode,
        'hard_metric_eligible': bool(hard_metric_eligible),
        'lesion_volume_bin': lesion_volume_bin,
        'hard_voxels': hard_voxels,
        'soft_support_voxels': soft_support_voxels,
        'voxel_volume_ml': voxel_volume_ml,
        'hard_target_volume_ml': hard_volume_ml,
        'soft_target_volume_ml': soft_volume_ml,
        'effective_target_volume_ml': float(effective_volume_ml),
    }


def locked_mask_metrics(prediction, target, spacing, probability_role='local'):
    intersection = int((prediction & target).sum())
    predicted = int(prediction.sum())
    target_voxels = int(target.sum())
    denominator = predicted + target_voxels
    dice = 2.0 * intersection / denominator if denominator else 1.0
    lesion_tp, lesion_fp, lesion_fn = pilot_lesion_counts(prediction, target)
    iou_threshold = 0.05 if probability_role == 'proposal' else 0.10
    components = pilot_target_component_iou_records(
        prediction, target, spacing
    ) if target_voxels > 0 else []
    component_tp = sum(
        float(item['maximum_component_iou']) >= iou_threshold
        for item in components
    )
    small_components = [
        item for item in components
        if item['component_tier'] in {
            'subvoxel_or_single_voxel', 'micro', 'tiny', 'small'
        }
    ]
    small_tp = sum(
        float(item['maximum_component_iou']) >= iou_threshold
        for item in small_components
    )
    return {
        'intersection_voxels': intersection,
        'predicted_voxels': predicted,
        'target_voxels': target_voxels,
        'dice': float(dice),
        'voxel_precision': intersection / max(predicted, 1),
        'voxel_recall': intersection / max(target_voxels, 1),
        'lesion_true_positive': int(lesion_tp),
        'lesion_false_positive': int(lesion_fp),
        'lesion_false_negative': int(lesion_fn),
        'component_true_positive': int(component_tp),
        'component_total': int(len(components)),
        'small_component_true_positive': int(small_tp),
        'small_component_total': int(len(small_components)),
        'components': components,
    }


def locked_aggregate_case_rows(frame):
    eligible = frame[frame['hard_metric_eligible'].astype(bool)].copy()
    positive = eligible[eligible['target_voxels'].astype(int) > 0].copy()
    tiny_small = positive[positive['lesion_volume_bin'].isin(['tiny', 'small'])]
    intersection = int(positive['intersection_voxels'].sum())
    predicted = int(positive['predicted_voxels'].sum())
    target = int(positive['target_voxels'].sum())
    lesion_tp = int(eligible['lesion_true_positive'].sum())
    lesion_fp = int(eligible['lesion_false_positive'].sum())
    lesion_fn = int(eligible['lesion_false_negative'].sum())
    precision = lesion_tp / max(lesion_tp + lesion_fp, 1)
    recall = lesion_tp / max(lesion_tp + lesion_fn, 1)
    lesion_f1 = (
        2.0 * precision * recall / max(precision + recall, 1e-12)
    )
    component_tp = int(positive['component_true_positive'].sum())
    component_total = int(positive['component_total'].sum())
    small_tp = int(positive['small_component_true_positive'].sum())
    small_total = int(positive['small_component_total'].sum())
    target_volume = float(positive['target_volume_ml'].sum())
    predicted_volume = float(positive['predicted_volume_ml'].sum())
    return {
        'subjects': int(len(frame)),
        'hard_metric_eligible_subjects': int(len(eligible)),
        'positive_subjects': int(len(positive)),
        'positive_mean_dice': float(positive['dice'].mean()) if len(positive) else 0.0,
        'positive_median_dice': float(positive['dice'].median()) if len(positive) else 0.0,
        'positive_global_dice': float(
            2.0 * intersection / max(predicted + target, 1)
        ),
        'lesion_precision': float(precision),
        'lesion_recall': float(recall),
        'lesion_f1': float(lesion_f1),
        'component_recall': component_tp / max(component_total, 1),
        'small_component_recall': small_tp / max(small_total, 1),
        'tiny_small_subject_mean_dice': float(
            tiny_small['dice'].mean()
        ) if len(tiny_small) else 0.0,
        'predicted_to_target_volume_ratio': (
            predicted_volume / max(target_volume, 1e-12)
        ),
        'empty_case_mean_false_positive_volume_ml': float(
            eligible.loc[
                eligible['target_voxels'].astype(int) == 0,
                'predicted_volume_ml',
            ].mean()
        ) if bool((eligible['target_voxels'].astype(int) == 0).any()) else 0.0,
        'intersection_voxels': intersection,
        'predicted_voxels_positive_cases': predicted,
        'target_voxels_positive_cases': target,
        'lesion_true_positive': lesion_tp,
        'lesion_false_positive': lesion_fp,
        'lesion_false_negative': lesion_fn,
        'component_true_positive': component_tp,
        'component_total': component_total,
        'small_component_true_positive': small_tp,
        'small_component_total': small_total,
    }


def locked_bootstrap_split(frame, split, replicates, seed):
    frame = frame.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    metric_names = [
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]
    observed = locked_aggregate_case_rows(frame)
    samples = {name: np.empty(replicates, dtype=np.float64) for name in metric_names}
    for replicate in range(replicates):
        indices = rng.integers(0, len(frame), size=len(frame))
        summary = locked_aggregate_case_rows(frame.iloc[indices])
        for name in metric_names:
            samples[name][replicate] = float(summary[name])
    rows = []
    for name in metric_names:
        values = samples[name]
        rows.append({
            'dev_split': split,
            'metric': name,
            'point_estimate': float(observed[name]),
            'ci_95_lower': float(np.quantile(values, 0.025)),
            'ci_95_upper': float(np.quantile(values, 0.975)),
            'bootstrap_standard_error': float(values.std(ddof=1)),
            'bootstrap_replicates': int(replicates),
            'bootstrap_seed': int(seed),
            'method': (
                'subject_level_nonparametric_percentile_fixed_frozen_policy'
            ),
        })
    return rows


def locked_existing_final_is_valid():
    if not LOCKED_FINAL_GATE_PATH.exists():
        return None
    gate = load_json(LOCKED_FINAL_GATE_PATH)
    expected_outputs = {
        'target_blind_prediction_gate', 'source_artifact_manifest',
        'prediction_audit', 'target_policy_audit', 'case_metrics',
        'component_metrics',
        'split_summary', 'lesion_bin_metrics', 'ensemble_member_metrics',
        'ensemble_diversity_metrics', 'bootstrap_intervals',
        'oof_to_locked_generalization', 'evaluation_report',
    }
    output_manifest = gate.get('output_artifact_manifest', {})
    source_manifest = gate.get('source_artifact_manifest', [])
    checks = {
        'evaluation_complete': gate.get('evaluation_complete') is True,
        'version_matches': gate.get('evaluation_version') == LOCKED_EVALUATION_VERSION,
        'revision_matches': gate.get('code_revision') == LOCKED_EVALUATION_CODE_REVISION,
        'local_threshold_matches': math.isclose(
            float(gate.get('frozen_local_threshold', float('nan'))),
            EXPECTED_FROZEN_LOCAL_THRESHOLD, rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_matches': math.isclose(
            float(gate.get('frozen_proposal_threshold', float('nan'))),
            EXPECTED_FROZEN_PROPOSAL_THRESHOLD, rel_tol=0.0, abs_tol=1e-12,
        ),
        'validation_and_test_evaluated_together': (
            gate.get('evaluated_splits') == ['val', 'test']
        ),
        'no_tuning': gate.get('post_freeze_tuning_performed') is False,
        'output_manifest_schema_exact': set(output_manifest) == expected_outputs,
        'source_manifest_complete': len(source_manifest) == 69,
        'source_predictions_reused': gate.get('source_predictions_reused') is True,
        'model_not_constructed': (
            gate.get('model_constructed_for_metric_reconciliation') is False
        ),
        'target_policy_derived_after_prediction_commit': (
            gate.get('target_policy_derived_after_prediction_commit') is True
        ),
        'recorded_final_checks_all_true': bool(
            gate.get('final_checks')
            and all(gate.get('final_checks', {}).values())
        ),
        'artifacts_stable': all(
            Path(record['path']).is_file()
            and file_sha256(Path(record['path'])) == record['sha256']
            for record in output_manifest.values()
        ),
        'sources_stable': all(
            Path(record['path']).is_file()
            and file_sha256(Path(record['path'])) == record['sha256']
            for record in source_manifest
        ),
    }
    locked_checks_or_raise(
        checks,
        'Existing locked-evaluation gate failed integrity verification; '
        'refusing to rerun or overwrite',
    )
    return gate


def run_locked_fivefold_evaluation():
    existing = locked_existing_final_is_valid()
    if existing is not None:
        summaries = existing['split_summaries']
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL LOCKED VALIDATION/TEST EVALUATION ALREADY COMPLETE')
        print('=' * 80)
        for split in ['val', 'test']:
            item = summaries[split]
            print(
                f'{split.upper():10s} mean/global Dice:             '
                f'{item["positive_mean_dice"]:.4f} / '
                f'{item["positive_global_dice"]:.4f}'
            )
        print(f'Final gate: {LOCKED_FINAL_GATE_PATH}')
        print('No checkpoint, prediction, target, or result artifact was rewritten.')
        return

    required_paths = [
        OOF_DEPLOYMENT_POLICY_PATH, OOF_AGGREGATION_GATE_PATH,
        MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
        MULTIMODAL_GATE_PATH,
    ]
    missing = [str(path) for path in required_paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(
            'Locked-evaluation prerequisites are missing:\n' + '\n'.join(missing)
        )
    policy = load_json(OOF_DEPLOYMENT_POLICY_PATH)
    oof_gate = load_json(OOF_AGGREGATION_GATE_PATH)
    policy_hash = file_sha256(OOF_DEPLOYMENT_POLICY_PATH)
    source_records = [
        locked_source_record(OOF_DEPLOYMENT_POLICY_PATH, 'deployment_policy'),
        locked_source_record(OOF_AGGREGATION_GATE_PATH, 'oof_freeze_gate'),
        locked_source_record(MULTIMODAL_MANIFEST_PATH, 'multimodal_manifest'),
        locked_source_record(FOLD_MANIFEST_PATH, 'fold_manifest'),
        locked_source_record(GRAPH_SCHEMA_PATH, 'graph_schema'),
        locked_source_record(MULTIMODAL_GATE_PATH, 'multimodal_gate'),
    ]
    checkpoints = sorted(
        policy.get('ensemble_checkpoints', []),
        key=lambda item: int(item['development_fold']),
    )
    for record in checkpoints:
        source_records.append(locked_source_record(
            record['best_checkpoint_path'], 'best_ema_checkpoint',
            record['development_fold'],
        ))
        graph_scaler = Path(resolve_gate_mapping(
            multimodal_gate['graph_scaler_paths'], record['development_fold']
        ))
        clinical_scaler = Path(resolve_gate_mapping(
            multimodal_gate['clinical_scaler_paths'], record['development_fold']
        ))
        source_records.append(locked_source_record(
            graph_scaler, 'fold_graph_scaler', record['development_fold']
        ))
        source_records.append(locked_source_record(
            clinical_scaler, 'fold_clinical_scaler', record['development_fold']
        ))
    local_threshold = float(policy.get('frozen_local_threshold', float('nan')))
    proposal_threshold = float(
        policy.get('frozen_proposal_threshold', float('nan'))
    )
    checkpoint_folds = tuple(
        int(record['development_fold']) for record in checkpoints
    )
    upstream_checks = {
        'oof_aggregation_complete': oof_gate.get('aggregation_complete') is True,
        'locked_evaluation_authorized': (
            oof_gate.get('locked_evaluation_authorized') is True
            and policy.get('locked_evaluation_authorized') is True
        ),
        'aggregation_version_matches': (
            oof_gate.get('aggregation_version') == EXPECTED_OOF_AGGREGATION_VERSION
            and policy.get('aggregation_version') == EXPECTED_OOF_AGGREGATION_VERSION
        ),
        'aggregation_code_revision_matches': (
            oof_gate.get('code_revision') == EXPECTED_OOF_CODE_REVISION
            and policy.get('code_revision') == EXPECTED_OOF_CODE_REVISION
        ),
        'policy_hash_matches_oof_output_manifest': (
            oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('path') == str(OOF_DEPLOYMENT_POLICY_PATH)
            and oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('sha256') == policy_hash
        ),
        'oof_final_checks_all_true': bool(
            oof_gate.get('final_checks')
            and all(oof_gate.get('final_checks', {}).values())
        ),
        'five_checkpoints_exact': checkpoint_folds == EXPECTED_ENSEMBLE_FOLDS,
        'ema_weight_state_exact': all(
            record.get('weight_state') == 'ema_model_state'
            for record in checkpoints
        ) and policy.get('checkpoint_weight_state') == 'ema_model_state',
        'checkpoint_hashes_match_policy': all(
            file_sha256(record['best_checkpoint_path'])
            == record['best_checkpoint_sha256']
            for record in checkpoints
        ),
        'checkpoint_parameters_match': all(
            int(record['parameter_count']) == EXPECTED_V22_PARAMETERS
            for record in checkpoints
        ),
        'local_threshold_exact': math.isclose(
            local_threshold, EXPECTED_FROZEN_LOCAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_exact': math.isclose(
            proposal_threshold, EXPECTED_FROZEN_PROPOSAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'probability_ensemble_rule_exact': (
            policy.get('ensemble_probability_rule')
            == 'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing_none': policy.get('postprocessing') == 'none',
        'tta_none': policy.get('test_time_augmentation') == 'none',
        'both_locked_splits_required': (
            policy.get('locked_evaluation_splits') == ['val', 'test']
        ),
        'sources_initially_stable': all(
            locked_sources_unchanged(source_records).values()
        ),
    }
    locked_checks_or_raise(
        upstream_checks, 'Frozen deployment-policy authorization failed'
    )

    manifest = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    manifest['subject_id'] = manifest['subject_id'].astype(str)
    locked_rows = manifest[
        manifest['dev_split'].astype(str).isin(['val', 'test'])
    ].copy().sort_values(['dev_split', 'subject_id'])
    split_counts = locked_rows['dev_split'].value_counts().to_dict()
    manifest_checks = {
        'locked_subject_count_is_46': len(locked_rows) == 46,
        'locked_subjects_unique': locked_rows['subject_id'].is_unique,
        'split_counts_exact': split_counts == EXPECTED_LOCKED_SPLITS,
        **locked_manifest_input_contract_checks(locked_rows),
    }
    locked_checks_or_raise(manifest_checks, 'Locked manifest audit failed')

    prediction_contract = {
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'deployment_policy_sha256': policy_hash,
        'checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_folds': list(checkpoint_folds),
        'weight_state': 'ema_model_state',
        'ensemble_rule': 'arithmetic_mean_of_five_fold_ema_probabilities',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'subject_ids': locked_rows['subject_id'].tolist(),
        'target_access_before_prediction_commit': False,
    }
    prediction_contract_hash = locked_canonical_hash(prediction_contract)
    LOCKED_PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)
    LOCKED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    if LOCKED_PREDICTION_GATE_PATH.exists():
        prediction_gate = load_json(LOCKED_PREDICTION_GATE_PATH)
        gate_checks = {
            'prediction_complete': prediction_gate.get('prediction_complete') is True,
            'contract_matches': (
                prediction_gate.get('prediction_contract_sha256')
                == prediction_contract_hash
            ),
            'all_46_subjects': prediction_gate.get('subjects_predicted') == 46,
            'locked_targets_not_read': (
                prediction_gate.get('locked_validation_targets_read') is False
                and prediction_gate.get('test_targets_read') is False
            ),
            'prediction_archives_stable': all(
                Path(record['path']).is_file()
                and file_sha256(record['path']) == record['sha256']
                for record in prediction_gate.get('prediction_manifest', [])
            ),
        }
        locked_checks_or_raise(
            gate_checks, 'Existing target-blind prediction gate rejected'
        )
        prediction_manifest = prediction_gate['prediction_manifest']
        print('\nVerified complete target-blind prediction gate; skipping inference.')
    else:
        if not torch.cuda.is_available():
            raise RuntimeError('Final five-model evaluation requires a CUDA GPU.')
        device = torch.device('cuda')
        set_determinism(PILOT_SEED + 9_000)
        gc.collect()
        torch.cuda.empty_cache()
        models = []
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL TARGET-BLIND FIVE-MODEL INFERENCE STARTING')
        print('=' * 80)
        print('Locked targets read:                False')
        print('Optimizer constructed / steps:      False / 0')
        for record in checkpoints:
            fold = int(record['development_fold'])
            checkpoint = torch.load(
                record['best_checkpoint_path'], map_location='cpu',
                weights_only=False,
            )
            checkpoint_checks = {
                'fold_matches': checkpoint.get('development_fold') == fold,
                'best_epoch_matches': int(checkpoint.get('epoch', -1))
                == int(record['best_epoch']),
                'parameter_count_matches': int(
                    checkpoint.get('parameter_count', -1)
                ) == EXPECTED_V22_PARAMETERS,
                'training_series_matches': (
                    checkpoint.get('replication_series_version')
                    == REPLICATION_SERIES_VERSION
                ),
                'training_revision_matches': (
                    checkpoint.get('code_revision') == PILOT_CODE_REVISION
                ),
                'checkpoint_policy_is_strict_numeric': (
                    checkpoint.get('checkpoint_selection_policy')
                    == 'strict_numerical_maximum'
                ),
                'ema_state_present': isinstance(
                    checkpoint.get('ema_model_state'), dict
                ),
            }
            locked_checks_or_raise(
                checkpoint_checks, f'Fold-{fold} checkpoint identity failed'
            )
            model = VascuFateNetV22(pilot_schema)
            if sum(parameter.numel() for parameter in model.parameters()) != (
                EXPECTED_V22_PARAMETERS
            ):
                raise RuntimeError(f'Fold-{fold}: model parameter count changed.')
            model.load_state_dict(checkpoint['ema_model_state'], strict=True)
            model = model.to(device).eval()
            for parameter in model.parameters():
                parameter.requires_grad_(False)
            if not all(
                bool(torch.isfinite(parameter).all().item())
                for parameter in model.parameters()
            ):
                raise RuntimeError(f'Fold-{fold}: EMA parameters are non-finite.')
            models.append((fold, model))
            del checkpoint
            print(f'Loaded fold-{fold} strict-best EMA checkpoint.')

        prior_index = (
            load_json(LOCKED_PREDICTION_INDEX_PATH)
            if LOCKED_PREDICTION_INDEX_PATH.exists() else {}
        )
        if prior_index and prior_index.get('prediction_contract_sha256') != (
            prediction_contract_hash
        ):
            raise RuntimeError(
                'Partial prediction index belongs to another contract; '
                'refusing to overwrite it.'
            )
        prediction_manifest = []
        for index, row in enumerate(locked_rows.to_dict('records'), start=1):
            subject_id = str(row['subject_id'])
            split = str(row['dev_split'])
            archive_path = locked_prediction_path(subject_id)
            existing_archive = locked_validate_prediction_archive(
                archive_path, prediction_contract_hash, subject_id
            )
            if existing_archive is not None:
                prediction_manifest.append(existing_archive)
                print(
                    f'Target-blind prediction {index:02d}/46: {subject_id} '
                    '(verified resume)'
                )
                continue
            local_sum = None
            proposal_sum = None
            fold_masks = []
            fold_presence = []
            fold_routed_volume = []
            for fold, model in models:
                global RUNTIME_FOLD
                RUNTIME_FOLD = fold
                result = locked_infer_case(model, row, device)
                local_probability = result['probability']
                proposal_probability = result['proposal_probability']
                if local_sum is None:
                    local_sum = np.zeros_like(local_probability, dtype=np.float32)
                    proposal_sum = np.zeros_like(
                        proposal_probability, dtype=np.float32
                    )
                local_sum += local_probability
                proposal_sum += proposal_probability
                fold_masks.append(local_probability >= local_threshold)
                fold_presence.append(result['presence_probability'])
                fold_routed_volume.append(result['routed_volume_ml'])
                del result, local_probability, proposal_probability
            ensemble_local_probability = local_sum / 5.0
            ensemble_proposal_probability = proposal_sum / 5.0
            ensemble_local = ensemble_local_probability >= local_threshold
            ensemble_proposal = (
                ensemble_proposal_probability >= proposal_threshold
            )
            shape = tuple(int(v) for v in ensemble_local.shape)
            fold_packed = np.stack([
                locked_pack_mask(mask) for mask in fold_masks
            ], axis=0)
            local_probability_hash = hashlib.sha256(
                np.ascontiguousarray(
                    ensemble_local_probability.astype('<f4')
                ).tobytes()
            ).hexdigest()
            proposal_probability_hash = hashlib.sha256(
                np.ascontiguousarray(
                    ensemble_proposal_probability.astype('<f4')
                ).tobytes()
            ).hexdigest()
            locked_atomic_npz(
                archive_path,
                subject_id=np.asarray(subject_id, dtype=np.str_),
                dev_split=np.asarray(split, dtype=np.str_),
                native_shape=np.asarray(shape, dtype=np.int64),
                contract_sha256=np.asarray(
                    prediction_contract_hash, dtype=np.str_
                ),
                fold_ids=np.asarray(EXPECTED_ENSEMBLE_FOLDS, dtype=np.int64),
                fold_local_masks_packed=fold_packed,
                ensemble_local_mask_packed=locked_pack_mask(ensemble_local),
                ensemble_proposal_mask_packed=locked_pack_mask(
                    ensemble_proposal
                ),
                fold_predicted_voxels=np.asarray([
                    int(mask.sum()) for mask in fold_masks
                ], dtype=np.int64),
                fold_presence_probabilities=np.asarray(
                    fold_presence, dtype=np.float32
                ),
                fold_routed_volume_ml=np.asarray(
                    fold_routed_volume, dtype=np.float32
                ),
                ensemble_local_probability_sha256=np.asarray(
                    local_probability_hash, dtype=np.str_
                ),
                ensemble_proposal_probability_sha256=np.asarray(
                    proposal_probability_hash, dtype=np.str_
                ),
            )
            manifest_row = locked_prediction_manifest_row(
                subject_id, split, archive_path, shape, fold_masks,
                ensemble_local, ensemble_proposal,
                ensemble_local_probability, ensemble_proposal_probability,
                fold_presence, fold_routed_volume,
            )
            prediction_manifest.append(manifest_row)
            atomic_write_json({
                'prediction_contract_sha256': prediction_contract_hash,
                'completed_subjects': len(prediction_manifest),
                'prediction_manifest': prediction_manifest,
                'locked_validation_targets_read': False,
                'test_targets_read': False,
            }, LOCKED_PREDICTION_INDEX_PATH)
            print(
                f'Target-blind prediction {index:02d}/46: {subject_id} '
                f'({split})'
            )
            del (
                local_sum, proposal_sum, fold_masks,
                ensemble_local_probability, ensemble_proposal_probability,
                ensemble_local, ensemble_proposal,
            )
            gc.collect()
            torch.cuda.empty_cache()

        
        normalized_manifest = []
        for row in locked_rows.to_dict('records'):
            subject_id = str(row['subject_id'])
            archive_path = locked_prediction_path(subject_id)
            locked_validate_prediction_archive(
                archive_path, prediction_contract_hash, subject_id
            )
            with np.load(archive_path, allow_pickle=False) as data:
                shape = tuple(int(v) for v in data['native_shape'].tolist())
                fold_masks = [
                    locked_unpack_mask(bits, shape)
                    for bits in data['fold_local_masks_packed']
                ]
                ensemble_local = locked_unpack_mask(
                    data['ensemble_local_mask_packed'], shape
                )
                ensemble_proposal = locked_unpack_mask(
                    data['ensemble_proposal_mask_packed'], shape
                )
                fold_presence = np.asarray(
                    data['fold_presence_probabilities'], dtype=np.float32
                )
                fold_routed = np.asarray(
                    data['fold_routed_volume_ml'], dtype=np.float32
                )
                probability_stats = {
                    'local_mean': float('nan'), 'local_max': float('nan'),
                    'proposal_mean': float('nan'), 'proposal_max': float('nan'),
                }
            pairwise = locked_pairwise_dice(fold_masks)
            vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
            union_voxels = int((vote_count > 0).sum())
            disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
            normalized_manifest.append({
                'subject_id': subject_id,
                'dev_split': str(row['dev_split']),
                'path': str(archive_path),
                'sha256': file_sha256(archive_path),
                'native_shape': list(shape),
                'ensemble_local_voxels': int(ensemble_local.sum()),
                'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
                'fold_local_predicted_voxels': [
                    int(mask.sum()) for mask in fold_masks
                ],
                'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
                'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
                'disagreement_union_fraction': float(
                    disagreement / max(union_voxels, 1)
                ),
                'mean_presence_probability': float(fold_presence.mean()),
                'mean_routed_volume_ml': float(fold_routed.mean()),
            })
        prediction_manifest = normalized_manifest
        prediction_gate = {
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
            'evaluation_version': LOCKED_EVALUATION_VERSION,
            'code_revision': LOCKED_EVALUATION_CODE_REVISION,
            'prediction_complete': True,
            'prediction_contract': prediction_contract,
            'prediction_contract_sha256': prediction_contract_hash,
            'subjects_predicted': len(prediction_manifest),
            'prediction_manifest': prediction_manifest,
            'ensemble_members': 5,
            'model_constructed_for_inference_only': True,
            'optimizer_constructed': False,
            'optimizer_steps': 0,
            'training_performed': False,
            'locked_validation_targets_read': False,
            'test_targets_read': False,
            'targets_opened_only_after_this_gate': True,
            'locked_manifest_checks': manifest_checks,
        }
        prediction_checks = {
            'all_46_predictions_present': len(prediction_manifest) == 46,
            'subjects_unique': len({
                item['subject_id'] for item in prediction_manifest
            }) == 46,
            'archives_stable': all(
                Path(item['path']).is_file()
                and file_sha256(item['path']) == item['sha256']
                for item in prediction_manifest
            ),
            'sources_still_unchanged': all(
                locked_sources_unchanged(source_records).values()
            ),
            'locked_manifest_checks_all_true': bool(
                manifest_checks and all(manifest_checks.values())
            ),
        }
        locked_checks_or_raise(
            prediction_checks, 'Target-blind prediction commit failed'
        )
        prediction_gate['checks'] = prediction_checks
        atomic_write_json(prediction_gate, LOCKED_PREDICTION_GATE_PATH)
        del models
        gc.collect()
        torch.cuda.empty_cache()

    
    
    print('\nTarget-blind prediction gate committed. Loading both locked target sets.')
    target_cache = {}
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
        target, soft, spacing = locked_load_target(row, shape)
        target_cache[subject_id] = {
            'target': target,
            'soft_nonzero_voxels': int((soft > 0).sum()),
            'spacing': spacing,
        }
    if len(target_cache) != 46:
        raise RuntimeError('Both locked target sets were not loaded completely.')

    case_rows = []
    component_rows = []
    member_rows = []
    diversity_rows = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
            fold_masks = [
                locked_unpack_mask(bits, shape)
                for bits in data['fold_local_masks_packed']
            ]
            ensemble_local = locked_unpack_mask(
                data['ensemble_local_mask_packed'], shape
            )
            ensemble_proposal = locked_unpack_mask(
                data['ensemble_proposal_mask_packed'], shape
            )
            fold_presence = np.asarray(
                data['fold_presence_probabilities'], dtype=np.float32
            )
            fold_routed = np.asarray(
                data['fold_routed_volume_ml'], dtype=np.float32
            )
        target = target_cache[subject_id]['target']
        spacing = target_cache[subject_id]['spacing']
        voxel_volume_ml = float(np.prod(spacing) / 1000.0)
        hard_eligible = as_bool(row['trial8_hard_metric_eligible'])
        lesion_bin = str(row['trial8_effective_lesion_volume_bin'])
        local_metrics = locked_mask_metrics(
            ensemble_local, target, spacing, 'local'
        )
        proposal_metrics = locked_mask_metrics(
            ensemble_proposal, target, spacing, 'proposal'
        )
        case_record = {
            'subject_id': subject_id,
            'dev_split': split,
            'lesion_volume_bin': lesion_bin,
            'hard_metric_eligible': bool(hard_eligible),
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': local_metrics['target_voxels'],
            'predicted_voxels': local_metrics['predicted_voxels'],
            'intersection_voxels': local_metrics['intersection_voxels'],
            'dice': local_metrics['dice'],
            'voxel_precision': local_metrics['voxel_precision'],
            'voxel_recall': local_metrics['voxel_recall'],
            'target_volume_ml': local_metrics['target_voxels'] * voxel_volume_ml,
            'predicted_volume_ml': (
                local_metrics['predicted_voxels'] * voxel_volume_ml
            ),
            'lesion_true_positive': local_metrics['lesion_true_positive'],
            'lesion_false_positive': local_metrics['lesion_false_positive'],
            'lesion_false_negative': local_metrics['lesion_false_negative'],
            'component_true_positive': local_metrics['component_true_positive'],
            'component_total': local_metrics['component_total'],
            'small_component_true_positive': (
                local_metrics['small_component_true_positive']
            ),
            'small_component_total': local_metrics['small_component_total'],
            'proposal_predicted_voxels': proposal_metrics['predicted_voxels'],
            'proposal_dice': proposal_metrics['dice'],
            'proposal_component_true_positive': (
                proposal_metrics['component_true_positive']
            ),
            'proposal_component_total': proposal_metrics['component_total'],
            'mean_presence_probability': float(fold_presence.mean()),
            'mean_routed_volume_ml': float(fold_routed.mean()),
            'soft_nonzero_voxels': target_cache[subject_id][
                'soft_nonzero_voxels'
            ],
            'prediction_archive_path': str(archive_path),
            'prediction_archive_sha256': file_sha256(archive_path),
        }
        for fold, mask in zip(EXPECTED_ENSEMBLE_FOLDS, fold_masks):
            fold_metrics = locked_mask_metrics(mask, target, spacing, 'local')
            case_record[f'fold_{fold}_dice'] = fold_metrics['dice']
            member_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'development_fold_model': fold,
                'hard_metric_eligible': bool(hard_eligible),
                'lesion_volume_bin': lesion_bin,
                'target_voxels': fold_metrics['target_voxels'],
                'predicted_voxels': fold_metrics['predicted_voxels'],
                'intersection_voxels': fold_metrics['intersection_voxels'],
                'dice': fold_metrics['dice'],
                'target_volume_ml': fold_metrics['target_voxels'] * voxel_volume_ml,
                'predicted_volume_ml': (
                    fold_metrics['predicted_voxels'] * voxel_volume_ml
                ),
                'lesion_true_positive': fold_metrics['lesion_true_positive'],
                'lesion_false_positive': fold_metrics['lesion_false_positive'],
                'lesion_false_negative': fold_metrics['lesion_false_negative'],
                'component_true_positive': fold_metrics['component_true_positive'],
                'component_total': fold_metrics['component_total'],
                'small_component_true_positive': (
                    fold_metrics['small_component_true_positive']
                ),
                'small_component_total': fold_metrics['small_component_total'],
            })
        pairwise = locked_pairwise_dice(fold_masks)
        vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
        union = int((vote_count > 0).sum())
        disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
        diversity_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
            'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
            'maximum_pairwise_fold_mask_dice': float(np.max(pairwise)),
            'prediction_union_voxels': union,
            'prediction_disagreement_voxels': disagreement,
            'disagreement_union_fraction': disagreement / max(union, 1),
            'ensemble_dice': local_metrics['dice'],
        })
        for component in local_metrics['components']:
            proposal_component = next(
                item for item in proposal_metrics['components']
                if int(item['component_id']) == int(component['component_id'])
            )
            component_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'component_id': int(component['component_id']),
                'component_tier': component['component_tier'],
                'component_voxels': int(component['component_voxels']),
                'component_volume_ml': float(component['component_volume_ml']),
                'local_maximum_iou': float(component['maximum_component_iou']),
                'local_detected_iou_ge_0p10': bool(
                    float(component['maximum_component_iou']) >= 0.10
                ),
                'proposal_maximum_iou': float(
                    proposal_component['maximum_component_iou']
                ),
                'proposal_detected_iou_ge_0p05': bool(
                    float(proposal_component['maximum_component_iou']) >= 0.05
                ),
            })
        case_rows.append(case_record)

    case_df = pd.DataFrame(case_rows).sort_values(['dev_split', 'subject_id'])
    component_df = pd.DataFrame(component_rows)
    member_case_df = pd.DataFrame(member_rows)
    diversity_df = pd.DataFrame(diversity_rows)
    split_summary_rows = []
    member_summary_rows = []
    bin_rows = []
    bootstrap_rows = []
    split_summaries = {}
    for split in ['val', 'test']:
        split_frame = case_df[case_df['dev_split'] == split].copy()
        summary = locked_aggregate_case_rows(split_frame)
        summary['dev_split'] = split
        split_summaries[split] = summary
        split_summary_rows.append(summary)
        bootstrap_rows.extend(locked_bootstrap_split(
            split_frame, split, LOCKED_BOOTSTRAP_REPLICATES,
            LOCKED_BOOTSTRAP_SEED + (0 if split == 'val' else 1),
        ))
        for lesion_bin in ['empty', 'tiny', 'small', 'medium', 'large', 'very_large']:
            bin_frame = split_frame[
                split_frame['lesion_volume_bin'] == lesion_bin
            ].copy()
            if bin_frame.empty:
                continue
            eligible_bin = bin_frame[
                bin_frame['hard_metric_eligible'].astype(bool)
            ]
            positive_bin = eligible_bin[
                eligible_bin['target_voxels'].astype(int) > 0
            ]
            bin_rows.append({
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'subjects': int(len(bin_frame)),
                'positive_subjects': int(len(positive_bin)),
                'mean_dice': float(positive_bin['dice'].mean())
                if len(positive_bin) else float('nan'),
                'median_dice': float(positive_bin['dice'].median())
                if len(positive_bin) else float('nan'),
                'pooled_global_dice': float(
                    2.0 * positive_bin['intersection_voxels'].sum()
                    / max(
                        positive_bin['predicted_voxels'].sum()
                        + positive_bin['target_voxels'].sum(), 1,
                    )
                ) if len(positive_bin) else float('nan'),
                'mean_target_volume_ml': float(
                    positive_bin['target_volume_ml'].mean()
                ) if len(positive_bin) else 0.0,
                'mean_predicted_volume_ml': float(
                    eligible_bin['predicted_volume_ml'].mean()
                ) if len(eligible_bin) else 0.0,
            })
        for fold in EXPECTED_ENSEMBLE_FOLDS:
            member_frame = member_case_df[
                (member_case_df['dev_split'] == split)
                & (member_case_df['development_fold_model'] == fold)
            ].copy()
            member_summary = locked_aggregate_case_rows(member_frame)
            member_summary.update({
                'dev_split': split,
                'model': f'fold_{fold}_ema',
                'development_fold_model': fold,
            })
            member_summary_rows.append(member_summary)
        ensemble_member_summary = summary.copy()
        ensemble_member_summary.update({
            'model': 'fivefold_probability_ensemble',
            'development_fold_model': -1,
        })
        member_summary_rows.append(ensemble_member_summary)

    split_summary_df = pd.DataFrame(split_summary_rows)
    bin_df = pd.DataFrame(bin_rows)
    member_summary_df = pd.DataFrame(member_summary_rows)
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    prediction_audit_df = pd.DataFrame(prediction_manifest)
    source_manifest_df = pd.DataFrame(source_records)
    oof_reference = policy['primary_oof_metrics']
    generalization_rows = []
    metric_mapping = {
        'positive_mean_dice': 'positive_case_mean_dice',
        'positive_median_dice': 'positive_case_median_dice',
        'positive_global_dice': 'pooled_global_dice',
        'lesion_f1': 'lesion_f1',
        'tiny_small_subject_mean_dice': (
            'tiny_small_subject_weighted_mean_dice'
        ),
        'component_recall': 'component_recall',
        'small_component_recall': 'small_component_recall',
    }
    for split in ['val', 'test']:
        for locked_metric, oof_metric in metric_mapping.items():
            locked_value = float(split_summaries[split][locked_metric])
            oof_value = float(oof_reference[oof_metric])
            generalization_rows.append({
                'dev_split': split,
                'metric': locked_metric,
                'development_oof_value': oof_value,
                'locked_value': locked_value,
                'absolute_change_from_oof': locked_value - oof_value,
                'relative_change_from_oof': (
                    (locked_value - oof_value) / abs(oof_value)
                    if abs(oof_value) > 1e-12 else float('nan')
                ),
                'interpretation': (
                    'confirmatory_transfer_diagnostic_not_a_tuning_signal'
                ),
            })
    generalization_df = pd.DataFrame(generalization_rows)
    ensemble_gain = {}
    for split in ['val', 'test']:
        members = member_summary_df[
            (member_summary_df['dev_split'] == split)
            & (member_summary_df['development_fold_model'] >= 0)
        ]
        ensemble_gain[split] = {
            'mean_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_mean_dice']
                - members['positive_mean_dice'].mean()
            ),
            'global_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_global_dice']
                - members['positive_global_dice'].mean()
            ),
            'lesion_f1_gain_over_mean_member': float(
                split_summaries[split]['lesion_f1']
                - members['lesion_f1'].mean()
            ),
        }

    final_checks = {
        **{
            f'locked_manifest_{name}': bool(passed)
            for name, passed in manifest_checks.items()
        },
        'prediction_gate_complete': load_json(
            LOCKED_PREDICTION_GATE_PATH
        ).get('prediction_complete') is True,
        'prediction_contract_matches': load_json(
            LOCKED_PREDICTION_GATE_PATH
        ).get('prediction_contract_sha256') == prediction_contract_hash,
        'all_46_cases_evaluated': len(case_df) == 46,
        'validation_has_23_cases': int(
            (case_df['dev_split'] == 'val').sum()
        ) == 23,
        'test_has_23_cases': int(
            (case_df['dev_split'] == 'test').sum()
        ) == 23,
        'all_subjects_unique': case_df['subject_id'].is_unique,
        'all_case_metrics_finite': bool(np.isfinite(case_df[[
            'dice', 'voxel_precision', 'voxel_recall',
            'target_volume_ml', 'predicted_volume_ml',
        ]].to_numpy(dtype=float)).all()),
        'both_split_summaries_finite': bool(np.isfinite(
            split_summary_df.select_dtypes(include=[np.number]).to_numpy()
        ).all()),
        'bootstrap_complete': len(bootstrap_df) == 20,
        'bootstrap_intervals_finite': bool(np.isfinite(bootstrap_df[[
            'point_estimate', 'ci_95_lower', 'ci_95_upper',
            'bootstrap_standard_error',
        ]].to_numpy(dtype=float)).all()),
        'five_members_plus_ensemble_per_split': len(member_summary_df) == 12,
        'generalization_rows_complete': len(generalization_df) == 14,
        'generalization_core_values_finite': bool(np.isfinite(
            generalization_df[[
                'development_oof_value', 'locked_value',
                'absolute_change_from_oof',
            ]].to_numpy(dtype=float)
        ).all()),
        'frozen_local_threshold_unchanged': local_threshold == 0.40,
        'frozen_proposal_threshold_unchanged': proposal_threshold == 0.35,
        'postprocessing_remained_none': policy['postprocessing'] == 'none',
        'tta_remained_none': policy['test_time_augmentation'] == 'none',
        'architecture_unchanged': True,
        'checkpoints_unchanged': all(
            locked_sources_unchanged(source_records).values()
        ),
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'post_freeze_tuning_not_performed': True,
        'validation_and_test_evaluated_in_one_frozen_execution': True,
    }
    locked_checks_or_raise(final_checks, 'Final locked evaluation failed')

    atomic_write_csv(source_manifest_df, LOCKED_SOURCE_MANIFEST_PATH)
    atomic_write_csv(prediction_audit_df, LOCKED_PREDICTION_AUDIT_PATH)
    atomic_write_csv(case_df, LOCKED_CASE_METRICS_PATH)
    atomic_write_csv(component_df, LOCKED_COMPONENT_METRICS_PATH)
    atomic_write_csv(split_summary_df, LOCKED_SPLIT_SUMMARY_PATH)
    atomic_write_csv(bin_df, LOCKED_BIN_METRICS_PATH)
    atomic_write_csv(member_summary_df, LOCKED_MEMBER_METRICS_PATH)
    atomic_write_csv(diversity_df, LOCKED_DIVERSITY_PATH)
    atomic_write_csv(bootstrap_df, LOCKED_BOOTSTRAP_PATH)
    atomic_write_csv(generalization_df, LOCKED_GENERALIZATION_PATH)
    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'selected_architecture': policy['selected_architecture'],
        'ensemble_members': 5,
        'checkpoint_weight_state': 'ema_model_state',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'development_oof_reference': oof_reference,
        'ensemble_gain_over_mean_member': ensemble_gain,
        'bootstrap_replicates': LOCKED_BOOTSTRAP_REPLICATES,
        'metric_scope': (
            'hard-metric-eligible positive cases for Dice; hard-metric-eligible '
            'cases for lesion detection; thresholds frozen from development OOF'
        ),
        'validation_used_for_tuning': False,
        'test_used_for_tuning': False,
        'post_freeze_tuning_performed': False,
        'scientific_interpretation': (
            'Validation and test are confirmatory locked evaluations. The '
            'development OOF estimate remains separately reported and is not '
            'pooled with either locked split as a primary performance estimate.'
        ),
    }
    atomic_write_json(report, LOCKED_REPORT_PATH)
    output_paths = {
        'target_blind_prediction_gate': LOCKED_PREDICTION_GATE_PATH,
        'source_artifact_manifest': LOCKED_SOURCE_MANIFEST_PATH,
        'prediction_audit': LOCKED_PREDICTION_AUDIT_PATH,
        'case_metrics': LOCKED_CASE_METRICS_PATH,
        'component_metrics': LOCKED_COMPONENT_METRICS_PATH,
        'split_summary': LOCKED_SPLIT_SUMMARY_PATH,
        'lesion_bin_metrics': LOCKED_BIN_METRICS_PATH,
        'ensemble_member_metrics': LOCKED_MEMBER_METRICS_PATH,
        'ensemble_diversity_metrics': LOCKED_DIVERSITY_PATH,
        'bootstrap_intervals': LOCKED_BOOTSTRAP_PATH,
        'oof_to_locked_generalization': LOCKED_GENERALIZATION_PATH,
        'evaluation_report': LOCKED_REPORT_PATH,
    }
    output_manifest = {
        name: {'path': str(path), 'sha256': file_sha256(path)}
        for name, path in output_paths.items()
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'evaluation_complete': True,
        'evaluated_splits': ['val', 'test'],
        'subjects_evaluated': {'val': 23, 'test': 23},
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'ensemble_members': 5,
        'ensemble_checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_weight_state': 'ema_model_state',
        'ensemble_probability_rule': (
            'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'prediction_contract_sha256': prediction_contract_hash,
        'prediction_gate_path': str(LOCKED_PREDICTION_GATE_PATH),
        'prediction_gate_sha256': file_sha256(LOCKED_PREDICTION_GATE_PATH),
        'source_artifact_manifest': source_records,
        'output_artifact_manifest': output_manifest,
        'final_checks': final_checks,
        'locked_validation_targets_read': True,
        'test_targets_read': True,
        'targets_read_only_after_target_blind_prediction_commit': True,
        'model_constructed_for_inference_only': True,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'post_freeze_tuning_performed': False,
        'further_model_or_threshold_tuning_authorized': False,
        'next_action': (
            'Report the frozen OOF, locked validation, and locked test results; '
            'perform no further model selection or threshold tuning.'
        ),
    }
    atomic_write_json(gate, LOCKED_FINAL_GATE_PATH)

    print('\nLocked split summary:')
    display(split_summary_df[[
        'dev_split', 'subjects', 'positive_subjects',
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]])
    print('\nLocked lesion-volume-bin metrics:')
    display(bin_df)
    print('\nLocked 95% descriptive bootstrap intervals:')
    display(bootstrap_df)
    print('\nEnsemble versus member models:')
    display(member_summary_df[[
        'dev_split', 'model', 'positive_mean_dice',
        'positive_global_dice', 'lesion_f1',
        'tiny_small_subject_mean_dice',
    ]])
    print('\nDevelopment OOF to locked-split generalization:')
    display(generalization_df[[
        'dev_split', 'metric', 'development_oof_value', 'locked_value',
        'absolute_change_from_oof',
    ]])
    print('\n' + '=' * 80)
    print('TRIAL8 FINAL LOCKED VALIDATION/TEST ENSEMBLE EVALUATION PASSED')
    print('=' * 80)
    for split in ['val', 'test']:
        item = split_summaries[split]
        print(
            f'{split.upper():10s} mean / median / global Dice: '
            f'{item["positive_mean_dice"]:.4f} / '
            f'{item["positive_median_dice"]:.4f} / '
            f'{item["positive_global_dice"]:.4f}'
        )
        print(
            f'{split.upper():10s} lesion F1 / component recall: '
            f'{item["lesion_f1"]:.4f} / '
            f'{item["component_recall"]:.4f}'
        )
    print('Frozen local / proposal threshold:  0.40 / 0.35')
    print('Postprocessing / TTA:               none / none')
    print('Post-freeze tuning performed:       False')
    print('Further tuning authorized:          False')
    print(f'Final report:                       {LOCKED_REPORT_PATH}')
    print(f'Final gate:                         {LOCKED_FINAL_GATE_PATH}')
    print('\nNext: report results and limitations; do not tune on these splits.')


def locked_optional_text(value):
    if value is None:
        return ''
    try:
        if bool(pd.isna(value)):
            return ''
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    return '' if text.lower() in {'nan', 'none', 'null'} else text


def locked_optional_bool(value):
    if value is None:
        return None
    try:
        if bool(pd.isna(value)):
            return None
    except (TypeError, ValueError):
        pass
    try:
        return bool(as_bool(value))
    except ValueError:
        return None


def locked_artifact_manifest_is_stable(manifest):
    if not isinstance(manifest, dict) or not manifest:
        return False
    return all(
        isinstance(record, dict)
        and Path(str(record.get('path', ''))).is_file()
        and file_sha256(Path(str(record['path']))) == record.get('sha256')
        for record in manifest.values()
    )


def locked_record_manifest_is_stable(records):
    if not isinstance(records, list) or not records:
        return False
    return all(
        isinstance(record, dict)
        and Path(str(record.get('path', ''))).is_file()
        and file_sha256(Path(str(record['path']))) == record.get('sha256')
        for record in records
    )


def run_locked_metric_reconciliation():
    existing = locked_existing_final_is_valid()
    if existing is not None:
        summaries = existing['split_summaries']
        print('\n' + '=' * 80)
        print('TRIAL8 FINAL LOCKED METRIC RECONCILIATION ALREADY COMPLETE')
        print('=' * 80)
        for split in ['val', 'test']:
            item = summaries[split]
            print(
                f'{split.upper():10s} mean/global Dice:             '
                f'{item["positive_mean_dice"]:.4f} / '
                f'{item["positive_global_dice"]:.4f}'
            )
        print(f'Final gate: {LOCKED_FINAL_GATE_PATH}')
        print('No checkpoint, prediction, target, or result artifact was rewritten.')
        return

    required_paths = [
        OOF_DEPLOYMENT_POLICY_PATH, OOF_AGGREGATION_GATE_PATH,
        MULTIMODAL_MANIFEST_PATH, FOLD_MANIFEST_PATH, GRAPH_SCHEMA_PATH,
        MULTIMODAL_GATE_PATH, LOCKED_PREDICTION_GATE_PATH,
        SOURCE_FLAWED_FINAL_GATE_PATH,
    ]
    missing = [str(path) for path in required_paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(
            'Metric-reconciliation prerequisites are missing:\n'
            + '\n'.join(missing)
        )

    policy = load_json(OOF_DEPLOYMENT_POLICY_PATH)
    oof_gate = load_json(OOF_AGGREGATION_GATE_PATH)
    source_prediction_gate = load_json(LOCKED_PREDICTION_GATE_PATH)
    source_flawed_gate = load_json(SOURCE_FLAWED_FINAL_GATE_PATH)
    policy_hash = file_sha256(OOF_DEPLOYMENT_POLICY_PATH)
    source_prediction_gate_hash = file_sha256(LOCKED_PREDICTION_GATE_PATH)
    source_flawed_gate_hash = file_sha256(SOURCE_FLAWED_FINAL_GATE_PATH)

    source_records = [
        locked_source_record(OOF_DEPLOYMENT_POLICY_PATH, 'deployment_policy'),
        locked_source_record(OOF_AGGREGATION_GATE_PATH, 'oof_freeze_gate'),
        locked_source_record(MULTIMODAL_MANIFEST_PATH, 'multimodal_manifest'),
        locked_source_record(FOLD_MANIFEST_PATH, 'fold_manifest'),
        locked_source_record(GRAPH_SCHEMA_PATH, 'graph_schema'),
        locked_source_record(MULTIMODAL_GATE_PATH, 'multimodal_gate'),
    ]
    checkpoints = sorted(
        policy.get('ensemble_checkpoints', []),
        key=lambda item: int(item['development_fold']),
    )
    for record in checkpoints:
        fold = int(record['development_fold'])
        source_records.append(locked_source_record(
            record['best_checkpoint_path'], 'best_ema_checkpoint', fold,
        ))
        graph_scaler = Path(resolve_gate_mapping(
            multimodal_gate['graph_scaler_paths'], fold
        ))
        clinical_scaler = Path(resolve_gate_mapping(
            multimodal_gate['clinical_scaler_paths'], fold
        ))
        source_records.append(locked_source_record(
            graph_scaler, 'fold_graph_scaler', fold
        ))
        source_records.append(locked_source_record(
            clinical_scaler, 'fold_clinical_scaler', fold
        ))
    base_source_count = len(source_records)
    source_records.extend([
        locked_source_record(
            LOCKED_PREDICTION_GATE_PATH, 'source_r2_prediction_gate'
        ),
        locked_source_record(
            SOURCE_FLAWED_FINAL_GATE_PATH,
            'source_r2_superseded_metric_gate',
        ),
    ])

    local_threshold = float(policy.get('frozen_local_threshold', float('nan')))
    proposal_threshold = float(
        policy.get('frozen_proposal_threshold', float('nan'))
    )
    checkpoint_folds = tuple(
        int(record['development_fold']) for record in checkpoints
    )
    upstream_checks = {
        'oof_aggregation_complete': oof_gate.get('aggregation_complete') is True,
        'locked_evaluation_authorized': (
            oof_gate.get('locked_evaluation_authorized') is True
            and policy.get('locked_evaluation_authorized') is True
        ),
        'aggregation_version_matches': (
            oof_gate.get('aggregation_version')
            == EXPECTED_OOF_AGGREGATION_VERSION
            and policy.get('aggregation_version')
            == EXPECTED_OOF_AGGREGATION_VERSION
        ),
        'aggregation_code_revision_matches': (
            oof_gate.get('code_revision') == EXPECTED_OOF_CODE_REVISION
            and policy.get('code_revision') == EXPECTED_OOF_CODE_REVISION
        ),
        'policy_hash_matches_oof_output_manifest': (
            oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('path') == str(OOF_DEPLOYMENT_POLICY_PATH)
            and oof_gate.get('output_artifact_manifest', {}).get(
                'frozen_deployment_policy', {}
            ).get('sha256') == policy_hash
        ),
        'oof_final_checks_all_true': bool(
            oof_gate.get('final_checks')
            and all(oof_gate.get('final_checks', {}).values())
        ),
        'five_checkpoints_exact': checkpoint_folds == EXPECTED_ENSEMBLE_FOLDS,
        'ema_weight_state_exact': all(
            record.get('weight_state') == 'ema_model_state'
            for record in checkpoints
        ) and policy.get('checkpoint_weight_state') == 'ema_model_state',
        'checkpoint_hashes_match_policy': all(
            file_sha256(record['best_checkpoint_path'])
            == record['best_checkpoint_sha256']
            for record in checkpoints
        ),
        'checkpoint_parameters_match': all(
            int(record['parameter_count']) == EXPECTED_V22_PARAMETERS
            for record in checkpoints
        ),
        'local_threshold_exact': math.isclose(
            local_threshold, EXPECTED_FROZEN_LOCAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'proposal_threshold_exact': math.isclose(
            proposal_threshold, EXPECTED_FROZEN_PROPOSAL_THRESHOLD,
            rel_tol=0.0, abs_tol=1e-12,
        ),
        'probability_ensemble_rule_exact': (
            policy.get('ensemble_probability_rule')
            == 'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing_none': policy.get('postprocessing') == 'none',
        'tta_none': policy.get('test_time_augmentation') == 'none',
        'both_locked_splits_required': (
            policy.get('locked_evaluation_splits') == ['val', 'test']
        ),
        'base_source_manifest_has_21_records': base_source_count == 21,
        'source_r2_output_artifacts_stable': locked_artifact_manifest_is_stable(
            source_flawed_gate.get('output_artifact_manifest', {})
        ),
        'source_r2_upstream_sources_stable': locked_record_manifest_is_stable(
            source_flawed_gate.get('source_artifact_manifest', [])
        ),
        'sources_initially_stable': all(
            locked_sources_unchanged(source_records).values()
        ),
    }
    locked_checks_or_raise(
        upstream_checks, 'Frozen reconciliation authorization failed'
    )

    manifest = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    manifest['subject_id'] = manifest['subject_id'].astype(str)
    locked_rows = manifest[
        manifest['dev_split'].astype(str).isin(['val', 'test'])
    ].copy().sort_values(['dev_split', 'subject_id']).reset_index(drop=True)
    split_counts = locked_rows['dev_split'].value_counts().to_dict()
    manifest_checks = {
        'locked_subject_count_is_46': len(locked_rows) == 46,
        'locked_subjects_unique': locked_rows['subject_id'].is_unique,
        'split_counts_exact': split_counts == EXPECTED_LOCKED_SPLITS,
        **locked_manifest_input_contract_checks(locked_rows),
    }
    locked_checks_or_raise(manifest_checks, 'Locked manifest audit failed')

    source_prediction_contract = {
        'evaluation_version': SOURCE_PREDICTION_VERSION,
        'code_revision': SOURCE_PREDICTION_CODE_REVISION,
        'deployment_policy_sha256': policy_hash,
        'checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_folds': list(checkpoint_folds),
        'weight_state': 'ema_model_state',
        'ensemble_rule': 'arithmetic_mean_of_five_fold_ema_probabilities',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'subject_ids': locked_rows['subject_id'].tolist(),
        'target_access_before_prediction_commit': False,
    }
    source_prediction_contract_hash = locked_canonical_hash(
        source_prediction_contract
    )
    source_gate_manifest = source_prediction_gate.get('prediction_manifest', [])
    source_gate_subjects = [
        str(record.get('subject_id', '')) for record in source_gate_manifest
    ]
    source_gate_records = {
        str(record.get('subject_id', '')): record
        for record in source_gate_manifest
    }
    source_r2_checks = {
        'prediction_gate_version_matches': (
            source_prediction_gate.get('evaluation_version')
            == SOURCE_PREDICTION_VERSION
        ),
        'prediction_gate_revision_matches': (
            source_prediction_gate.get('code_revision')
            == SOURCE_PREDICTION_CODE_REVISION
        ),
        'prediction_complete': (
            source_prediction_gate.get('prediction_complete') is True
        ),
        'prediction_contract_hash_matches': (
            source_prediction_gate.get('prediction_contract_sha256')
            == source_prediction_contract_hash
        ),
        'stored_prediction_contract_hashes_exactly': (
            locked_canonical_hash(
                source_prediction_gate.get('prediction_contract', {})
            ) == source_prediction_contract_hash
        ),
        'all_46_subjects_predicted': (
            int(source_prediction_gate.get('subjects_predicted', -1)) == 46
        ),
        'prediction_manifest_has_46_records': len(source_gate_manifest) == 46,
        'prediction_manifest_subjects_unique': (
            len(set(source_gate_subjects)) == 46
        ),
        'prediction_manifest_subject_set_exact': (
            set(source_gate_subjects) == set(locked_rows['subject_id'])
        ),
        'prediction_gate_checks_all_true': bool(
            source_prediction_gate.get('checks')
            and all(source_prediction_gate.get('checks', {}).values())
        ),
        'targets_were_sealed_during_prediction': (
            source_prediction_gate.get('locked_validation_targets_read') is False
            and source_prediction_gate.get('test_targets_read') is False
        ),
        'source_final_gate_version_matches': (
            source_flawed_gate.get('evaluation_version')
            == SOURCE_PREDICTION_VERSION
        ),
        'source_final_gate_revision_matches': (
            source_flawed_gate.get('code_revision')
            == SOURCE_PREDICTION_CODE_REVISION
        ),
        'source_final_gate_complete': (
            source_flawed_gate.get('evaluation_complete') is True
        ),
        'source_final_checks_all_true': bool(
            source_flawed_gate.get('final_checks')
            and all(source_flawed_gate.get('final_checks', {}).values())
        ),
        'source_final_prediction_contract_matches': (
            source_flawed_gate.get('prediction_contract_sha256')
            == source_prediction_contract_hash
        ),
        'source_failure_signature_test_positive_count_zero': (
            int(source_flawed_gate.get('split_summaries', {}).get(
                'test', {}
            ).get('positive_subjects', -1)) == 0
        ),
        'source_validation_has_positive_cases': (
            int(source_flawed_gate.get('split_summaries', {}).get(
                'val', {}
            ).get('positive_subjects', 0)) > 0
        ),
        'source_prediction_gate_hash_unchanged': (
            file_sha256(LOCKED_PREDICTION_GATE_PATH)
            == source_prediction_gate_hash
        ),
        'source_flawed_gate_hash_unchanged': (
            file_sha256(SOURCE_FLAWED_FINAL_GATE_PATH)
            == source_flawed_gate_hash
        ),
    }
    locked_checks_or_raise(
        source_r2_checks, 'Frozen R2 prediction source rejected'
    )

    prediction_manifest = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        gate_record = source_gate_records[subject_id]
        gate_path_text = (
            gate_record.get('path') or gate_record.get('prediction_path') or ''
        )
        validated = locked_validate_prediction_archive(
            archive_path, source_prediction_contract_hash, subject_id
        )
        if validated is None:
            raise RuntimeError(
                f'{subject_id}: frozen R2 prediction archive is missing.'
            )
        with np.load(archive_path, allow_pickle=False) as data:
            archive_split = scalar_string(data['dev_split'])
            shape = tuple(int(v) for v in data['native_shape'].tolist())
            fold_ids = tuple(int(v) for v in data['fold_ids'].tolist())
            fold_masks = [
                locked_unpack_mask(bits, shape)
                for bits in data['fold_local_masks_packed']
            ]
            ensemble_local = locked_unpack_mask(
                data['ensemble_local_mask_packed'], shape
            )
            ensemble_proposal = locked_unpack_mask(
                data['ensemble_proposal_mask_packed'], shape
            )
            fold_voxels = np.asarray(
                data['fold_predicted_voxels'], dtype=np.int64
            )
            fold_presence = np.asarray(
                data['fold_presence_probabilities'], dtype=np.float64
            )
            fold_routed = np.asarray(
                data['fold_routed_volume_ml'], dtype=np.float64
            )
            local_probability_hash = scalar_string(
                data['ensemble_local_probability_sha256']
            )
            proposal_probability_hash = scalar_string(
                data['ensemble_proposal_probability_sha256']
            )
        gate_archive_hash = (
            gate_record.get('sha256')
            or gate_record.get('prediction_sha256')
        )
        archive_checks = {
            'manifest_path_matches_frozen_archive': (
                Path(gate_path_text) == archive_path
            ),
            'manifest_hash_matches_archive': (
                gate_archive_hash == file_sha256(archive_path)
            ),
            'archive_split_matches_manifest': archive_split == split,
            'fold_ids_exact': fold_ids == EXPECTED_ENSEMBLE_FOLDS,
            'five_fold_masks': len(fold_masks) == 5,
            'fold_voxel_counts_shape_exact': fold_voxels.shape == (5,),
            'fold_voxel_counts_match_masks': bool(np.array_equal(
                fold_voxels,
                np.asarray([int(mask.sum()) for mask in fold_masks]),
            )),
            'presence_shape_exact': fold_presence.shape == (5,),
            'routed_volume_shape_exact': fold_routed.shape == (5,),
            'presence_values_finite': bool(np.isfinite(fold_presence).all()),
            'routed_volume_values_finite_nonnegative': bool(
                np.isfinite(fold_routed).all() and (fold_routed >= 0).all()
            ),
            'probability_hashes_well_formed': (
                len(local_probability_hash) == 64
                and len(proposal_probability_hash) == 64
            ),
        }
        locked_checks_or_raise(
            archive_checks, f'{subject_id}: frozen R2 archive audit failed'
        )
        pairwise = locked_pairwise_dice(fold_masks)
        vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
        union_voxels = int((vote_count > 0).sum())
        disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
        prediction_manifest.append({
            'subject_id': subject_id,
            'dev_split': split,
            'path': str(archive_path),
            'sha256': file_sha256(archive_path),
            'native_shape': list(shape),
            'ensemble_local_voxels': int(ensemble_local.sum()),
            'ensemble_proposal_voxels': int(ensemble_proposal.sum()),
            'fold_local_predicted_voxels': [
                int(mask.sum()) for mask in fold_masks
            ],
            'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
            'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
            'disagreement_union_fraction': float(
                disagreement / max(union_voxels, 1)
            ),
            'mean_presence_probability': float(fold_presence.mean()),
            'mean_routed_volume_ml': float(fold_routed.mean()),
            'source_prediction_contract_sha256': (
                source_prediction_contract_hash
            ),
        })
        source_records.append(locked_source_record(
            archive_path, 'source_r2_prediction_archive'
        ))

    prediction_checks = {
        'all_46_archives_verified': len(prediction_manifest) == 46,
        'archive_subjects_unique': len({
            item['subject_id'] for item in prediction_manifest
        }) == 46,
        'archive_subject_set_exact': {
            item['subject_id'] for item in prediction_manifest
        } == set(locked_rows['subject_id']),
        'source_manifest_has_69_records': len(source_records) == 69,
        'all_source_records_stable_before_target_access': all(
            locked_sources_unchanged(source_records).values()
        ),
        'source_and_reconciliation_roots_are_distinct': (
            SOURCE_PREDICTION_ROOT.resolve() != LOCKED_OUTPUT_ROOT.resolve()
        ),
    }
    locked_checks_or_raise(
        prediction_checks, 'Frozen prediction archive commit verification failed'
    )

    LOCKED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print('\n' + '=' * 80)
    print('TRIAL8 R3 READ-ONLY LOCKED METRIC RECONCILIATION STARTING')
    print('=' * 80)
    print('Frozen R2 prediction archives:       46 / 46 verified')
    print('Inference repeated:                  False')
    print('Model / optimizer constructed:       False / False')
    print('Predictions or checkpoints rewritten: False')
    print('Thresholds changed:                  False')
    print('Loading both locked target sets only after source-gate verification.')

    target_cache = {}
    policy_audit_rows = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
        target, soft, spacing = locked_load_target(row, shape)
        derived = locked_derive_target_policy(target, soft, spacing)
        manifest_hard = locked_optional_bool(
            row.get('trial8_hard_metric_eligible')
        )
        manifest_bin = locked_optional_text(
            row.get('trial8_effective_lesion_volume_bin')
        )
        manifest_mode = locked_optional_text(
            row.get('trial8_supervision_mode')
        )
        hard_match = (
            manifest_hard is not None
            and manifest_hard == derived['hard_metric_eligible']
        )
        bin_match = manifest_bin == derived['lesion_volume_bin']
        mode_match = manifest_mode == derived['supervision_mode']
        target_cache[subject_id] = {
            'target': target,
            'soft_nonzero_voxels': derived['soft_support_voxels'],
            'spacing': spacing,
            'policy': derived,
        }
        policy_audit_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'derived_supervision_mode': derived['supervision_mode'],
            'derived_hard_metric_eligible': (
                derived['hard_metric_eligible']
            ),
            'derived_lesion_volume_bin': derived['lesion_volume_bin'],
            'hard_voxels': derived['hard_voxels'],
            'soft_support_voxels': derived['soft_support_voxels'],
            'hard_target_volume_ml': derived['hard_target_volume_ml'],
            'soft_target_volume_ml': derived['soft_target_volume_ml'],
            'effective_target_volume_ml': (
                derived['effective_target_volume_ml']
            ),
            'manifest_supervision_mode': manifest_mode,
            'manifest_hard_metric_eligible': manifest_hard,
            'manifest_lesion_volume_bin': manifest_bin,
            'supervision_mode_matches_manifest': bool(mode_match),
            'hard_metric_eligibility_matches_manifest': bool(hard_match),
            'lesion_volume_bin_matches_manifest': bool(bin_match),
            'manifest_policy_used_for_scoring': False,
            'target_policy_derived_after_prediction_commit': True,
        })
    if len(target_cache) != 46:
        raise RuntimeError('Both locked target sets were not loaded completely.')
    target_policy_df = pd.DataFrame(policy_audit_rows).sort_values(
        ['dev_split', 'subject_id']
    )
    validation_policy = target_policy_df[
        target_policy_df['dev_split'] == 'val'
    ]
    test_policy = target_policy_df[target_policy_df['dev_split'] == 'test']
    test_hard_positive_count = int((
        test_policy['derived_hard_metric_eligible'].astype(bool)
        & (test_policy['hard_voxels'].astype(int) > 0)
    ).sum())
    target_policy_checks = {
        'all_46_target_policies_derived': len(target_policy_df) == 46,
        'validation_has_23_derived_policies': len(validation_policy) == 23,
        'test_has_23_derived_policies': len(test_policy) == 23,
        'validation_declared_supervision_modes_reproduced': bool(
            validation_policy.loc[
                validation_policy['manifest_supervision_mode'].astype(str) != '',
                'supervision_mode_matches_manifest',
            ].all()
        ),
        'validation_hard_eligibility_reproduced_exactly': bool(
            validation_policy[
                'hard_metric_eligibility_matches_manifest'
            ].all()
        ),
        'validation_lesion_bins_reproduced_exactly': bool(
            validation_policy['lesion_volume_bin_matches_manifest'].all()
        ),
        'test_contains_target_derived_hard_positive_cases': (
            test_hard_positive_count > 0
        ),
        'manifest_policy_not_used_for_scoring': not bool(
            target_policy_df['manifest_policy_used_for_scoring'].any()
        ),
    }
    locked_checks_or_raise(
        target_policy_checks,
        'Target-derived policy reconciliation failed before metric calculation',
    )

    case_rows = []
    component_rows = []
    member_rows = []
    diversity_rows = []
    for row in locked_rows.to_dict('records'):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = locked_prediction_path(subject_id)
        with np.load(archive_path, allow_pickle=False) as data:
            shape = tuple(int(v) for v in data['native_shape'].tolist())
            fold_masks = [
                locked_unpack_mask(bits, shape)
                for bits in data['fold_local_masks_packed']
            ]
            ensemble_local = locked_unpack_mask(
                data['ensemble_local_mask_packed'], shape
            )
            ensemble_proposal = locked_unpack_mask(
                data['ensemble_proposal_mask_packed'], shape
            )
            fold_presence = np.asarray(
                data['fold_presence_probabilities'], dtype=np.float32
            )
            fold_routed = np.asarray(
                data['fold_routed_volume_ml'], dtype=np.float32
            )
        target = target_cache[subject_id]['target']
        spacing = target_cache[subject_id]['spacing']
        derived = target_cache[subject_id]['policy']
        voxel_volume_ml = derived['voxel_volume_ml']
        hard_eligible = derived['hard_metric_eligible']
        lesion_bin = derived['lesion_volume_bin']
        local_metrics = locked_mask_metrics(
            ensemble_local, target, spacing, 'local'
        )
        proposal_metrics = locked_mask_metrics(
            ensemble_proposal, target, spacing, 'proposal'
        )
        case_record = {
            'subject_id': subject_id,
            'dev_split': split,
            'supervision_mode': derived['supervision_mode'],
            'lesion_volume_bin': lesion_bin,
            'hard_metric_eligible': bool(hard_eligible),
            'metric_policy_source': 'target_derived_after_prediction_commit',
            'local_threshold': local_threshold,
            'proposal_threshold': proposal_threshold,
            'target_voxels': local_metrics['target_voxels'],
            'predicted_voxels': local_metrics['predicted_voxels'],
            'intersection_voxels': local_metrics['intersection_voxels'],
            'dice': local_metrics['dice'],
            'voxel_precision': local_metrics['voxel_precision'],
            'voxel_recall': local_metrics['voxel_recall'],
            'target_volume_ml': local_metrics['target_voxels'] * voxel_volume_ml,
            'predicted_volume_ml': (
                local_metrics['predicted_voxels'] * voxel_volume_ml
            ),
            'lesion_true_positive': local_metrics['lesion_true_positive'],
            'lesion_false_positive': local_metrics['lesion_false_positive'],
            'lesion_false_negative': local_metrics['lesion_false_negative'],
            'component_true_positive': local_metrics['component_true_positive'],
            'component_total': local_metrics['component_total'],
            'small_component_true_positive': (
                local_metrics['small_component_true_positive']
            ),
            'small_component_total': local_metrics['small_component_total'],
            'proposal_predicted_voxels': proposal_metrics['predicted_voxels'],
            'proposal_dice': proposal_metrics['dice'],
            'proposal_component_true_positive': (
                proposal_metrics['component_true_positive']
            ),
            'proposal_component_total': proposal_metrics['component_total'],
            'mean_presence_probability': float(fold_presence.mean()),
            'mean_routed_volume_ml': float(fold_routed.mean()),
            'soft_nonzero_voxels': derived['soft_support_voxels'],
            'prediction_archive_path': str(archive_path),
            'prediction_archive_sha256': file_sha256(archive_path),
        }
        for fold, mask in zip(EXPECTED_ENSEMBLE_FOLDS, fold_masks):
            fold_metrics = locked_mask_metrics(mask, target, spacing, 'local')
            case_record[f'fold_{fold}_dice'] = fold_metrics['dice']
            member_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'development_fold_model': fold,
                'hard_metric_eligible': bool(hard_eligible),
                'lesion_volume_bin': lesion_bin,
                'target_voxels': fold_metrics['target_voxels'],
                'predicted_voxels': fold_metrics['predicted_voxels'],
                'intersection_voxels': fold_metrics['intersection_voxels'],
                'dice': fold_metrics['dice'],
                'target_volume_ml': (
                    fold_metrics['target_voxels'] * voxel_volume_ml
                ),
                'predicted_volume_ml': (
                    fold_metrics['predicted_voxels'] * voxel_volume_ml
                ),
                'lesion_true_positive': fold_metrics['lesion_true_positive'],
                'lesion_false_positive': fold_metrics['lesion_false_positive'],
                'lesion_false_negative': fold_metrics['lesion_false_negative'],
                'component_true_positive': (
                    fold_metrics['component_true_positive']
                ),
                'component_total': fold_metrics['component_total'],
                'small_component_true_positive': (
                    fold_metrics['small_component_true_positive']
                ),
                'small_component_total': fold_metrics['small_component_total'],
            })
        pairwise = locked_pairwise_dice(fold_masks)
        vote_count = np.stack(fold_masks, axis=0).sum(axis=0)
        union = int((vote_count > 0).sum())
        disagreement = int(((vote_count > 0) & (vote_count < 5)).sum())
        diversity_rows.append({
            'subject_id': subject_id,
            'dev_split': split,
            'mean_pairwise_fold_mask_dice': float(np.mean(pairwise)),
            'minimum_pairwise_fold_mask_dice': float(np.min(pairwise)),
            'maximum_pairwise_fold_mask_dice': float(np.max(pairwise)),
            'prediction_union_voxels': union,
            'prediction_disagreement_voxels': disagreement,
            'disagreement_union_fraction': disagreement / max(union, 1),
            'ensemble_dice': local_metrics['dice'],
        })
        for component in local_metrics['components']:
            proposal_component = next(
                item for item in proposal_metrics['components']
                if int(item['component_id']) == int(component['component_id'])
            )
            component_rows.append({
                'subject_id': subject_id,
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'component_id': int(component['component_id']),
                'component_tier': component['component_tier'],
                'component_voxels': int(component['component_voxels']),
                'component_volume_ml': float(component['component_volume_ml']),
                'local_maximum_iou': float(component['maximum_component_iou']),
                'local_detected_iou_ge_0p10': bool(
                    float(component['maximum_component_iou']) >= 0.10
                ),
                'proposal_maximum_iou': float(
                    proposal_component['maximum_component_iou']
                ),
                'proposal_detected_iou_ge_0p05': bool(
                    float(proposal_component['maximum_component_iou']) >= 0.05
                ),
            })
        case_rows.append(case_record)

    case_df = pd.DataFrame(case_rows).sort_values(['dev_split', 'subject_id'])
    component_df = pd.DataFrame(component_rows)
    member_case_df = pd.DataFrame(member_rows)
    diversity_df = pd.DataFrame(diversity_rows)
    split_summary_rows = []
    member_summary_rows = []
    bin_rows = []
    bootstrap_rows = []
    split_summaries = {}
    for split in ['val', 'test']:
        split_frame = case_df[case_df['dev_split'] == split].copy()
        summary = locked_aggregate_case_rows(split_frame)
        summary['dev_split'] = split
        split_summaries[split] = summary
        split_summary_rows.append(summary)
        bootstrap_rows.extend(locked_bootstrap_split(
            split_frame, split, LOCKED_BOOTSTRAP_REPLICATES,
            LOCKED_BOOTSTRAP_SEED + (0 if split == 'val' else 1),
        ))
        for lesion_bin in [
            'empty', 'tiny', 'small', 'medium', 'large', 'very_large'
        ]:
            bin_frame = split_frame[
                split_frame['lesion_volume_bin'] == lesion_bin
            ].copy()
            if bin_frame.empty:
                continue
            eligible_bin = bin_frame[
                bin_frame['hard_metric_eligible'].astype(bool)
            ]
            positive_bin = eligible_bin[
                eligible_bin['target_voxels'].astype(int) > 0
            ]
            bin_rows.append({
                'dev_split': split,
                'lesion_volume_bin': lesion_bin,
                'subjects': int(len(bin_frame)),
                'positive_subjects': int(len(positive_bin)),
                'mean_dice': float(positive_bin['dice'].mean())
                if len(positive_bin) else float('nan'),
                'median_dice': float(positive_bin['dice'].median())
                if len(positive_bin) else float('nan'),
                'pooled_global_dice': float(
                    2.0 * positive_bin['intersection_voxels'].sum()
                    / max(
                        positive_bin['predicted_voxels'].sum()
                        + positive_bin['target_voxels'].sum(), 1,
                    )
                ) if len(positive_bin) else float('nan'),
                'mean_target_volume_ml': float(
                    positive_bin['target_volume_ml'].mean()
                ) if len(positive_bin) else 0.0,
                'mean_predicted_volume_ml': float(
                    eligible_bin['predicted_volume_ml'].mean()
                ) if len(eligible_bin) else 0.0,
            })
        for fold in EXPECTED_ENSEMBLE_FOLDS:
            member_frame = member_case_df[
                (member_case_df['dev_split'] == split)
                & (member_case_df['development_fold_model'] == fold)
            ].copy()
            member_summary = locked_aggregate_case_rows(member_frame)
            member_summary.update({
                'dev_split': split,
                'model': f'fold_{fold}_ema',
                'development_fold_model': fold,
            })
            member_summary_rows.append(member_summary)
        ensemble_member_summary = summary.copy()
        ensemble_member_summary.update({
            'model': 'fivefold_probability_ensemble',
            'development_fold_model': -1,
        })
        member_summary_rows.append(ensemble_member_summary)

    split_summary_df = pd.DataFrame(split_summary_rows)
    bin_df = pd.DataFrame(bin_rows)
    member_summary_df = pd.DataFrame(member_summary_rows)
    bootstrap_df = pd.DataFrame(bootstrap_rows)
    prediction_audit_df = pd.DataFrame(prediction_manifest)
    source_manifest_df = pd.DataFrame(source_records)
    oof_reference = policy['primary_oof_metrics']
    generalization_rows = []
    metric_mapping = {
        'positive_mean_dice': 'positive_case_mean_dice',
        'positive_median_dice': 'positive_case_median_dice',
        'positive_global_dice': 'pooled_global_dice',
        'lesion_f1': 'lesion_f1',
        'tiny_small_subject_mean_dice': (
            'tiny_small_subject_weighted_mean_dice'
        ),
        'component_recall': 'component_recall',
        'small_component_recall': 'small_component_recall',
    }
    for split in ['val', 'test']:
        for locked_metric, oof_metric in metric_mapping.items():
            locked_value = float(split_summaries[split][locked_metric])
            oof_value = float(oof_reference[oof_metric])
            generalization_rows.append({
                'dev_split': split,
                'metric': locked_metric,
                'development_oof_value': oof_value,
                'locked_value': locked_value,
                'absolute_change_from_oof': locked_value - oof_value,
                'relative_change_from_oof': (
                    (locked_value - oof_value) / abs(oof_value)
                    if abs(oof_value) > 1e-12 else float('nan')
                ),
                'interpretation': (
                    'confirmatory_transfer_diagnostic_not_a_tuning_signal'
                ),
            })
    generalization_df = pd.DataFrame(generalization_rows)
    ensemble_gain = {}
    for split in ['val', 'test']:
        members = member_summary_df[
            (member_summary_df['dev_split'] == split)
            & (member_summary_df['development_fold_model'] >= 0)
        ]
        ensemble_gain[split] = {
            'mean_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_mean_dice']
                - members['positive_mean_dice'].mean()
            ),
            'global_dice_gain_over_mean_member': float(
                split_summaries[split]['positive_global_dice']
                - members['positive_global_dice'].mean()
            ),
            'lesion_f1_gain_over_mean_member': float(
                split_summaries[split]['lesion_f1']
                - members['lesion_f1'].mean()
            ),
        }

    source_validation_summary = source_flawed_gate['split_summaries']['val']
    validation_float_metrics = [
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]
    validation_integer_metrics = [
        'subjects', 'hard_metric_eligible_subjects', 'positive_subjects',
        'intersection_voxels', 'predicted_voxels_positive_cases',
        'target_voxels_positive_cases', 'lesion_true_positive',
        'lesion_false_positive', 'lesion_false_negative',
        'component_true_positive', 'component_total',
        'small_component_true_positive', 'small_component_total',
    ]
    validation_reproduction_checks = {
        f'validation_{name}_reproduced': math.isclose(
            float(split_summaries['val'][name]),
            float(source_validation_summary[name]),
            rel_tol=0.0, abs_tol=1e-12,
        )
        for name in validation_float_metrics
    }
    validation_reproduction_checks.update({
        f'validation_{name}_reproduced': (
            int(split_summaries['val'][name])
            == int(source_validation_summary[name])
        )
        for name in validation_integer_metrics
    })

    final_checks = {
        **{
            f'upstream_{name}': bool(passed)
            for name, passed in upstream_checks.items()
        },
        **{
            f'locked_manifest_{name}': bool(passed)
            for name, passed in manifest_checks.items()
        },
        **{
            f'source_r2_{name}': bool(passed)
            for name, passed in source_r2_checks.items()
        },
        **{
            f'prediction_{name}': bool(passed)
            for name, passed in prediction_checks.items()
        },
        **{
            f'target_policy_{name}': bool(passed)
            for name, passed in target_policy_checks.items()
        },
        **validation_reproduction_checks,
        'all_46_cases_evaluated': len(case_df) == 46,
        'validation_has_23_cases': int(
            (case_df['dev_split'] == 'val').sum()
        ) == 23,
        'test_has_23_cases': int(
            (case_df['dev_split'] == 'test').sum()
        ) == 23,
        'test_positive_count_comes_from_targets': (
            int(split_summaries['test']['positive_subjects'])
            == test_hard_positive_count
            and test_hard_positive_count > 0
        ),
        'all_subjects_unique': case_df['subject_id'].is_unique,
        'all_case_metrics_finite': bool(np.isfinite(case_df[[
            'dice', 'voxel_precision', 'voxel_recall',
            'target_volume_ml', 'predicted_volume_ml',
        ]].to_numpy(dtype=float)).all()),
        'both_split_summaries_finite': bool(np.isfinite(
            split_summary_df.select_dtypes(include=[np.number]).to_numpy()
        ).all()),
        'bootstrap_complete': len(bootstrap_df) == 20,
        'bootstrap_intervals_finite': bool(np.isfinite(bootstrap_df[[
            'point_estimate', 'ci_95_lower', 'ci_95_upper',
            'bootstrap_standard_error',
        ]].to_numpy(dtype=float)).all()),
        'five_members_plus_ensemble_per_split': len(member_summary_df) == 12,
        'generalization_rows_complete': len(generalization_df) == 14,
        'generalization_core_values_finite': bool(np.isfinite(
            generalization_df[[
                'development_oof_value', 'locked_value',
                'absolute_change_from_oof',
            ]].to_numpy(dtype=float)
        ).all()),
        'source_prediction_contract_unchanged': (
            file_sha256(LOCKED_PREDICTION_GATE_PATH)
            == source_prediction_gate_hash
        ),
        'source_flawed_gate_unchanged': (
            file_sha256(SOURCE_FLAWED_FINAL_GATE_PATH)
            == source_flawed_gate_hash
        ),
        'all_source_predictions_and_checkpoints_unchanged': all(
            locked_sources_unchanged(source_records).values()
        ),
        'frozen_local_threshold_unchanged': local_threshold == 0.40,
        'frozen_proposal_threshold_unchanged': proposal_threshold == 0.35,
        'postprocessing_remained_none': policy['postprocessing'] == 'none',
        'tta_remained_none': policy['test_time_augmentation'] == 'none',
        'architecture_unchanged': True,
        'model_not_constructed': True,
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'inference_not_repeated': True,
        'training_not_performed': True,
        'post_freeze_tuning_not_performed': True,
        'validation_and_test_reconciled_in_one_frozen_execution': True,
    }
    locked_checks_or_raise(final_checks, 'Final locked reconciliation failed')

    atomic_write_csv(source_manifest_df, LOCKED_SOURCE_MANIFEST_PATH)
    atomic_write_csv(prediction_audit_df, LOCKED_PREDICTION_AUDIT_PATH)
    atomic_write_csv(target_policy_df, LOCKED_TARGET_POLICY_AUDIT_PATH)
    atomic_write_csv(case_df, LOCKED_CASE_METRICS_PATH)
    atomic_write_csv(component_df, LOCKED_COMPONENT_METRICS_PATH)
    atomic_write_csv(split_summary_df, LOCKED_SPLIT_SUMMARY_PATH)
    atomic_write_csv(bin_df, LOCKED_BIN_METRICS_PATH)
    atomic_write_csv(member_summary_df, LOCKED_MEMBER_METRICS_PATH)
    atomic_write_csv(diversity_df, LOCKED_DIVERSITY_PATH)
    atomic_write_csv(bootstrap_df, LOCKED_BOOTSTRAP_PATH)
    atomic_write_csv(generalization_df, LOCKED_GENERALIZATION_PATH)
    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'source_prediction_version': SOURCE_PREDICTION_VERSION,
        'source_prediction_code_revision': SOURCE_PREDICTION_CODE_REVISION,
        'source_prediction_contract_sha256': (
            source_prediction_contract_hash
        ),
        'selected_architecture': policy['selected_architecture'],
        'ensemble_members': 5,
        'checkpoint_weight_state': 'ema_model_state',
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'development_oof_reference': oof_reference,
        'ensemble_gain_over_mean_member': ensemble_gain,
        'bootstrap_replicates': LOCKED_BOOTSTRAP_REPLICATES,
        'metric_policy': {
            'source': 'target_derived_after_prediction_commit',
            'hard_and_soft': 'hard_metric_eligible',
            'soft_only_subvoxel': 'excluded_from_hard_metrics',
            'true_empty': 'eligible_for_lesion_detection_metrics',
            'volume_bins_ml': {
                'tiny': '[0,1)', 'small': '[1,5)',
                'medium': '[5,25)', 'large': '[25,75)',
                'very_large': '[75,infinity)',
            },
            'validation_policy_reproduced_exactly': True,
            'test_manifest_policy_used_for_scoring': False,
        },
        'metric_scope': (
            'hard-metric-eligible positive cases for Dice; hard-metric-eligible '
            'cases for lesion detection; thresholds frozen from development OOF'
        ),
        'source_r2_test_metrics_status': (
            'superseded_due_to_sealed_manifest_metric_eligibility'
        ),
        'source_predictions_reused': True,
        'inference_repeated': False,
        'validation_used_for_tuning': False,
        'test_used_for_tuning': False,
        'post_freeze_tuning_performed': False,
        'scientific_interpretation': (
            'Validation and test are confirmatory locked evaluations. R3 '
            'changes only post-commit metric eligibility accounting and '
            'reproduces the complete R2 validation metrics exactly. The '
            'development OOF estimate remains separately reported.'
        ),
    }
    atomic_write_json(report, LOCKED_REPORT_PATH)
    output_paths = {
        'target_blind_prediction_gate': LOCKED_PREDICTION_GATE_PATH,
        'source_artifact_manifest': LOCKED_SOURCE_MANIFEST_PATH,
        'prediction_audit': LOCKED_PREDICTION_AUDIT_PATH,
        'target_policy_audit': LOCKED_TARGET_POLICY_AUDIT_PATH,
        'case_metrics': LOCKED_CASE_METRICS_PATH,
        'component_metrics': LOCKED_COMPONENT_METRICS_PATH,
        'split_summary': LOCKED_SPLIT_SUMMARY_PATH,
        'lesion_bin_metrics': LOCKED_BIN_METRICS_PATH,
        'ensemble_member_metrics': LOCKED_MEMBER_METRICS_PATH,
        'ensemble_diversity_metrics': LOCKED_DIVERSITY_PATH,
        'bootstrap_intervals': LOCKED_BOOTSTRAP_PATH,
        'oof_to_locked_generalization': LOCKED_GENERALIZATION_PATH,
        'evaluation_report': LOCKED_REPORT_PATH,
    }
    output_manifest = {
        name: {'path': str(path), 'sha256': file_sha256(path)}
        for name, path in output_paths.items()
    }
    gate = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'evaluation_version': LOCKED_EVALUATION_VERSION,
        'code_revision': LOCKED_EVALUATION_CODE_REVISION,
        'evaluation_complete': True,
        'evaluated_splits': ['val', 'test'],
        'subjects_evaluated': {'val': 23, 'test': 23},
        'frozen_local_threshold': local_threshold,
        'frozen_proposal_threshold': proposal_threshold,
        'ensemble_members': 5,
        'ensemble_checkpoint_sha256': [
            record['best_checkpoint_sha256'] for record in checkpoints
        ],
        'checkpoint_weight_state': 'ema_model_state',
        'ensemble_probability_rule': (
            'arithmetic_mean_of_five_fold_ema_probabilities'
        ),
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'split_summaries': split_summaries,
        'prediction_contract_sha256': source_prediction_contract_hash,
        'prediction_gate_path': str(LOCKED_PREDICTION_GATE_PATH),
        'prediction_gate_sha256': source_prediction_gate_hash,
        'source_superseded_metric_gate_path': str(
            SOURCE_FLAWED_FINAL_GATE_PATH
        ),
        'source_superseded_metric_gate_sha256': source_flawed_gate_hash,
        'source_artifact_manifest': source_records,
        'output_artifact_manifest': output_manifest,
        'final_checks': final_checks,
        'locked_validation_targets_read': True,
        'test_targets_read': True,
        'targets_read_only_after_target_blind_prediction_commit': True,
        'target_policy_derived_after_prediction_commit': True,
        'validation_policy_reproduced_exactly': True,
        'test_manifest_policy_used_for_scoring': False,
        'source_predictions_reused': True,
        'inference_repeated': False,
        'model_constructed_for_metric_reconciliation': False,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'post_freeze_tuning_performed': False,
        'source_r2_test_metrics_superseded': True,
        'further_model_or_threshold_tuning_authorized': False,
        'next_action': (
            'Report the frozen OOF and reconciled locked validation/test '
            'results; perform no further model selection or threshold tuning.'
        ),
    }
    atomic_write_json(gate, LOCKED_FINAL_GATE_PATH)

    print('\nTarget-derived policy audit by split:')
    display(target_policy_df.groupby('dev_split', as_index=False).agg(
        subjects=('subject_id', 'count'),
        hard_metric_eligible=('derived_hard_metric_eligible', 'sum'),
        hard_positive_cases=('hard_voxels', lambda values: int((values > 0).sum())),
        supervision_mode_matches=('supervision_mode_matches_manifest', 'sum'),
        hard_eligibility_matches=(
            'hard_metric_eligibility_matches_manifest', 'sum'
        ),
        lesion_bin_matches=('lesion_volume_bin_matches_manifest', 'sum'),
    ))
    print('\nReconciled locked split summary:')
    display(split_summary_df[[
        'dev_split', 'subjects', 'positive_subjects',
        'positive_mean_dice', 'positive_median_dice',
        'positive_global_dice', 'lesion_precision', 'lesion_recall',
        'lesion_f1', 'component_recall', 'small_component_recall',
        'tiny_small_subject_mean_dice',
        'predicted_to_target_volume_ratio',
    ]])
    print('\nLocked lesion-volume-bin metrics:')
    display(bin_df)
    print('\nLocked 95% descriptive bootstrap intervals:')
    display(bootstrap_df)
    print('\nEnsemble versus member models:')
    display(member_summary_df[[
        'dev_split', 'model', 'positive_mean_dice',
        'positive_global_dice', 'lesion_f1',
        'tiny_small_subject_mean_dice',
    ]])
    print('\nDevelopment OOF to locked-split generalization:')
    display(generalization_df[[
        'dev_split', 'metric', 'development_oof_value', 'locked_value',
        'absolute_change_from_oof',
    ]])
    print('\n' + '=' * 80)
    print('TRIAL8 FINAL LOCKED METRIC RECONCILIATION PASSED')
    print('=' * 80)
    print('Frozen R2 predictions reused:        True')
    print('Inference / training repeated:       False / False')
    print('Validation policy reproduced:        True')
    print('R2 all-zero test metrics superseded: True')
    for split in ['val', 'test']:
        item = split_summaries[split]
        print(
            f'{split.upper():10s} mean / median / global Dice: '
            f'{item["positive_mean_dice"]:.4f} / '
            f'{item["positive_median_dice"]:.4f} / '
            f'{item["positive_global_dice"]:.4f}'
        )
        print(
            f'{split.upper():10s} lesion F1 / component recall: '
            f'{item["lesion_f1"]:.4f} / '
            f'{item["component_recall"]:.4f}'
        )
    print('Frozen local / proposal threshold:  0.40 / 0.35')
    print('Postprocessing / TTA:               none / none')
    print('Post-freeze tuning performed:       False')
    print('Further tuning authorized:          False')
    print(f'Final report:                       {LOCKED_REPORT_PATH}')
    print(f'Final gate:                         {LOCKED_FINAL_GATE_PATH}')
    print('\nNext: report results and limitations; do not tune on these splits.')


run_locked_metric_reconciliation()


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path






RETROSPECTIVE_VERSION = 'v1_locked_readonly_comparison'
RETROSPECTIVE_CODE_REVISION = (
    'r1_all_checkpoint_state_digest_fixed_policy_two_phase'
)
RETROSPECTIVE_SEED = 20261014
RETROSPECTIVE_LOCAL_THRESHOLD = 0.40
RETROSPECTIVE_PROPOSAL_THRESHOLD = 0.35
RETROSPECTIVE_BOOTSTRAP_REPLICATES = 10_000
RETROSPECTIVE_WEIGHT_STATES = ('ema_model_state', 'model_state')
RETROSPECTIVE_EXPECTED_SPLITS = {'val': 23, 'test': 23}
RETROSPECTIVE_EXPECTED_V22_PARAMETERS = 16_870_679
RETROSPECTIVE_EXPECTED_R8_PARAMETERS = 16_885_385
RETROSPECTIVE_R8_GATE_BOUND = 0.20
RETROSPECTIVE_R8_HIDDEN_CHANNELS = 16

RETROSPECTIVE_REQUIRED_RUNTIME_SYMBOLS = [
    'PROJECT_ROOT', 'R2_VERSION', 'MODEL_VERSION',
    'MULTIMODAL_MANIFEST_PATH', 'MULTIMODAL_GATE_PATH',
    'LOCKED_FINAL_GATE_PATH', 'LOCKED_CASE_METRICS_PATH',
    'pilot_schema', 'multimodal_gate', 'VascuFateNetV22',
    'ResidualBlock3D', 'ConfidenceGate', 'group_count',
    'GLOBAL_CANVAS_SHAPE', 'LOCAL_PATCH_SIZE', 'PILOT_WINDOW_WEIGHT',
    'pilot_global_cache', 'pilot_local_from_cache',
    'pilot_stack_sample', 'pilot_sliding_starts', 'pilot_window_sample',
    'sample_global_to_local', 'locked_load_target_free_case',
    'locked_load_target', 'locked_derive_target_policy',
    'locked_mask_metrics', 'locked_aggregate_case_rows',
    'locked_pack_mask', 'locked_unpack_mask', 'locked_atomic_npz',
    'locked_manifest_input_contract_checks', 'locked_checks_or_raise',
    'locked_canonical_hash', 'file_sha256', 'load_json', 'to_builtin',
    'set_determinism', 'as_bool', 'scalar_string',
    'np', 'pd', 'torch', 'nn', 'F', 'nib', 'display',
]
RETROSPECTIVE_BOOTSTRAP_MISSING = [
    name for name in RETROSPECTIVE_REQUIRED_RUNTIME_SYMBOLS
    if name not in globals()
]
if RETROSPECTIVE_BOOTSTRAP_MISSING:
    raise RuntimeError(
        'TRIAL8 ALL-CHECKPOINT AUDIT RUNTIME BOOTSTRAP REQUIRED. Run the '
        'complete current trial8_v2_2_r5_locked_metric_reconciliation_r3.py '
        'block once in this Colab runtime, then rerun this block. The completed '
        'R3 block is read-only and must not retrain or rewrite checkpoints. '
        'Missing runtime symbols: '
        + ', '.join(RETROSPECTIVE_BOOTSTRAP_MISSING)
    )

RETROSPECTIVE_ROOT = (
    PROJECT_ROOT / 'evaluation'
    / 'trial8_all_checkpoint_retrospective_audit'
    / RETROSPECTIVE_VERSION
)
RETROSPECTIVE_PREDICTION_ROOT = (
    RETROSPECTIVE_ROOT / 'target_blind_predictions'
)
RETROSPECTIVE_INVENTORY_PATH = (
    RETROSPECTIVE_ROOT / 'checkpoint_state_inventory.csv'
)
RETROSPECTIVE_CANDIDATE_PATH = (
    RETROSPECTIVE_ROOT / 'unique_inference_candidates.csv'
)
RETROSPECTIVE_EXCLUSION_PATH = (
    RETROSPECTIVE_ROOT / 'excluded_checkpoint_states.csv'
)
RETROSPECTIVE_CONTRACT_PATH = (
    RETROSPECTIVE_ROOT / 'retrospective_prediction_contract.json'
)
RETROSPECTIVE_PREDICTION_GATE_PATH = (
    RETROSPECTIVE_ROOT / 'target_blind_prediction_gate.json'
)
RETROSPECTIVE_TARGET_POLICY_PATH = (
    RETROSPECTIVE_ROOT / 'target_derived_policy_audit.csv'
)
RETROSPECTIVE_CASE_METRICS_PATH = (
    RETROSPECTIVE_ROOT / 'checkpoint_case_metrics.csv'
)
RETROSPECTIVE_SPLIT_SUMMARY_PATH = (
    RETROSPECTIVE_ROOT / 'checkpoint_split_summary.csv'
)
RETROSPECTIVE_RANKING_PATH = (
    RETROSPECTIVE_ROOT / 'exploratory_rankings.csv'
)
RETROSPECTIVE_BOOTSTRAP_PATH = (
    RETROSPECTIVE_ROOT / 'paired_bootstrap_vs_official_ensemble.csv'
)
RETROSPECTIVE_FAMILY_PATH = (
    RETROSPECTIVE_ROOT / 'architecture_family_summary.csv'
)
RETROSPECTIVE_REPORT_PATH = (
    RETROSPECTIVE_ROOT / 'retrospective_report.json'
)
RETROSPECTIVE_FINAL_GATE_PATH = (
    RETROSPECTIVE_ROOT / 'retrospective_audit_gate.json'
)


def retrospective_now():
    return datetime.now(timezone.utc).isoformat()


def retrospective_json_text(record):
    return json.dumps(
        to_builtin(record), indent=2, sort_keys=True, allow_nan=False
    ) + '\n'


def retrospective_write_once_bytes(path, payload):
    
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = bytes(payload)
    if path.exists():
        if path.read_bytes() != payload:
            raise RuntimeError(
                f'Immutable retrospective artifact conflicts with this run: {path}'
            )
        return False
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'wb') as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)
    return True


def retrospective_write_once_json(record, path):
    return retrospective_write_once_bytes(
        path, retrospective_json_text(record).encode('utf-8')
    )


def retrospective_write_once_csv(frame, path):
    text = frame.to_csv(index=False, lineterminator='\n')
    return retrospective_write_once_bytes(path, text.encode('utf-8'))


def retrospective_state_digest(state):
    
    digest = hashlib.sha256()
    if not isinstance(state, dict) or not state:
        raise RuntimeError('Checkpoint state is not a non-empty mapping.')
    for name in sorted(state):
        value = state[name]
        if not torch.is_tensor(value):
            raise RuntimeError(f'Non-tensor entry in state_dict: {name}')
        tensor = value.detach().cpu().contiguous()
        digest.update(str(name).encode('utf-8'))
        digest.update(str(tensor.dtype).encode('utf-8'))
        digest.update(np.asarray(tensor.shape, dtype=np.int64).tobytes())
        digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


def retrospective_state_parameter_like_count(state):
    return int(sum(
        int(value.numel()) for value in state.values()
        if torch.is_tensor(value)
    ))


def retrospective_path_family(path):
    text = str(path).lower()
    if 'v2_2_r5_final_fivefold_strictbest' in text:
        return 'R5_final_fivefold'
    if 'global_local_component_diagnostic_r5' in text:
        return 'R5_original_fold0'
    if 'local_contrastive' in text or '_r7' in text:
        return 'R7_local_contrastive'
    if 'dual_resolution' in text or '_r8' in text:
        return 'R8_dual_resolution_satellite'
    if 'component_rescue' in text or '_r9' in text:
        return 'R9_component_rescue'
    if 'vascufate_net_v2' in text:
        return 'earlier_V2_family'
    if 'trial8' in text:
        return 'earlier_Trial8_family'
    return 'unclassified'


def retrospective_checkpoint_role(path):
    name = Path(path).name.lower()
    if name == 'best_checkpoint.pt':
        return 'best'
    if name == 'last_checkpoint.pt':
        return 'last'
    if 'diagnostic_best' in name:
        return 'diagnostic_best'
    if 'best' in name:
        return 'other_best'
    if 'last' in name:
        return 'other_last'
    return 'other'


def retrospective_resolve_fold(checkpoint, path):
    for key in ['development_fold', 'fold', 'heldout_fold']:
        value = checkpoint.get(key)
        try:
            value = int(value)
        except (TypeError, ValueError):
            continue
        if value in range(5):
            return value, f'metadata:{key}'
    matches = re.findall(r'(?:fold|fold_)([0-4])', str(path).lower())
    if matches:
        return int(matches[-1]), 'path'
    family = retrospective_path_family(path)
    if family in {
        'R5_original_fold0', 'R7_local_contrastive',
        'R8_dual_resolution_satellite', 'R9_component_rescue',
    }:
        return 0, 'family_contract'
    return -1, 'unresolved'






class RetrospectiveNativeSatelliteProposalHead(nn.Module):
    def __init__(
        self, local_channels=32,
        hidden=RETROSPECTIVE_R8_HIDDEN_CHANNELS,
    ):
        super().__init__()
        self.project = nn.Sequential(
            nn.Conv3d(local_channels + 2, hidden, 1, bias=False),
            nn.GroupNorm(group_count(hidden), hidden),
            nn.GELU(),
        )
        self.refine = ResidualBlock3D(hidden, hidden, dropout=0.05)
        self.output = nn.Conv3d(hidden, 1, 1)

    def forward(self, local_feature, coarse_proposal, arterial_local):
        feature = self.project(torch.cat([
            local_feature, torch.sigmoid(coarse_proposal), arterial_local,
        ], dim=1))
        return self.output(self.refine(feature))


class RetrospectiveVascuFateNetR8(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.base = VascuFateNetV22(schema)
        self.satellite_head = RetrospectiveNativeSatelliteProposalHead()
        self.satellite_gate = ConfidenceGate(
            confidence_dim=5, output_dim=1,
            bound=RETROSPECTIVE_R8_GATE_BOUND, hidden_dim=32,
        )
        self._decoder_feature = None
        self._decoder_hook = self.base.local_model.decoder1.register_forward_hook(
            self._capture_decoder_feature
        )
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def _capture_decoder_feature(self, module, inputs, output):
        self._decoder_feature = output


class RetrospectiveNativeComponentRescueHead(nn.Module):
    def __init__(self, image_channels=15, decoder_channels=32):
        super().__init__()
        self.image_stem = nn.Sequential(
            nn.Conv3d(image_channels, 8, 3, padding=1, bias=False),
            nn.GroupNorm(group_count(8), 8), nn.GELU(),
        )
        self.context_stem = nn.Sequential(
            nn.Conv3d(decoder_channels + 2, 12, 1, bias=False),
            nn.GroupNorm(group_count(12), 12), nn.GELU(),
        )
        self.fusion = nn.Sequential(
            nn.Conv3d(20, 24, 1, bias=False),
            nn.GroupNorm(group_count(24), 24), nn.GELU(),
            ResidualBlock3D(24, 24, dropout=0.10),
            nn.Dropout3d(0.10),
        )
        self.center_output = nn.Conv3d(24, 1, 1)
        self.extent_output = nn.Conv3d(24, 1, 1)
        with torch.no_grad():
            self.center_output.bias.fill_(-2.2)
            self.extent_output.bias.fill_(-2.2)

    def forward(
        self, image, decoder_feature, coarse_proposal, arterial_local,
    ):
        image_feature = self.image_stem(image)
        context_feature = self.context_stem(torch.cat([
            decoder_feature, torch.sigmoid(coarse_proposal), arterial_local,
        ], dim=1))
        feature = self.fusion(torch.cat([
            image_feature, context_feature,
        ], dim=1))
        center_logits = self.center_output(feature)
        extent_logits = self.extent_output(feature)
        rescue_probability = torch.sqrt(torch.clamp(
            torch.sigmoid(center_logits) * torch.sigmoid(extent_logits),
            min=0.0, max=1.0,
        ))
        return center_logits, extent_logits, rescue_probability


class RetrospectiveVascuFateNetR9(nn.Module):
    def __init__(self, schema):
        super().__init__()
        self.base = VascuFateNetV22(schema)
        self.rescue_head = RetrospectiveNativeComponentRescueHead()
        self._decoder_feature = None
        self._decoder_hook = self.base.local_model.decoder1.register_forward_hook(
            self._capture_decoder_feature
        )
        self.direct_graph_to_logit_residual = False
        self.expert_routing = False

    def _capture_decoder_feature(self, module, inputs, output):
        self._decoder_feature = output


def retrospective_model(architecture):
    if architecture == 'VascuFateNetV22':
        return VascuFateNetV22(pilot_schema)
    if architecture == 'VascuFateNetR8':
        return RetrospectiveVascuFateNetR8(pilot_schema)
    if architecture == 'VascuFateNetR9':
        return RetrospectiveVascuFateNetR9(pilot_schema)
    raise ValueError(f'Unsupported retrospective architecture: {architecture}')


def retrospective_architecture_keys():
    models = {
        'VascuFateNetV22': retrospective_model('VascuFateNetV22'),
        'VascuFateNetR8': retrospective_model('VascuFateNetR8'),
        'VascuFateNetR9': retrospective_model('VascuFateNetR9'),
    }
    result = {
        name: set(model.state_dict()) for name, model in models.items()
    }
    counts = {
        name: int(sum(parameter.numel() for parameter in model.parameters()))
        for name, model in models.items()
    }
    for model in models.values():
        if hasattr(model, '_decoder_hook'):
            model._decoder_hook.remove()
    del models
    if counts['VascuFateNetV22'] != RETROSPECTIVE_EXPECTED_V22_PARAMETERS:
        raise RuntimeError('Runtime V2.2 parameter count changed.')
    if counts['VascuFateNetR8'] != RETROSPECTIVE_EXPECTED_R8_PARAMETERS:
        raise RuntimeError('Reconstructed R8 parameter count changed.')
    return result, counts


RETROSPECTIVE_ARCHITECTURE_KEYS, RETROSPECTIVE_ARCHITECTURE_COUNTS = (
    retrospective_architecture_keys()
)


def retrospective_classify_state(state):
    keys = set(state) if isinstance(state, dict) else set()
    exact = [
        name for name, expected in RETROSPECTIVE_ARCHITECTURE_KEYS.items()
        if keys == expected
    ]
    if len(exact) == 1:
        return exact[0], True, ''
    if any(name.startswith('rescue_head.') for name in keys):
        return (
            'R9_schema_mismatch', False,
            'R9-like state keys do not exactly match the reconstructed R9 schema',
        )
    if any(
        name.startswith('satellite_head.')
        or name.startswith('satellite_gate.')
        for name in keys
    ):
        return (
            'R8_schema_mismatch', False,
            'R8-like state keys do not exactly match the reconstructed R8 schema',
        )
    if keys:
        return (
            'unsupported_architecture', False,
            'state_dict key schema is not V2.2, R8, or R9',
        )
    return 'invalid_state', False, 'state_dict is empty or invalid'






def retrospective_inventory():
    checkpoint_root = PROJECT_ROOT / 'checkpoints'
    checkpoint_paths = sorted(
        path for path in checkpoint_root.rglob('*.pt') if path.is_file()
    )
    if not checkpoint_paths:
        raise RuntimeError(f'No checkpoint files found under {checkpoint_root}.')

    rows = []
    source_records = []
    unique = {}
    for index, path in enumerate(checkpoint_paths, start=1):
        path_hash = file_sha256(path)
        stat = path.stat()
        source_records.append({
            'path': str(path),
            'sha256': path_hash,
            'size_bytes': int(stat.st_size),
            'mtime_ns': int(stat.st_mtime_ns),
        })
        print(
            f'Checkpoint inventory {index:02d}/{len(checkpoint_paths):02d}: '
            f'{path.relative_to(checkpoint_root)}'
        )
        try:
            checkpoint = torch.load(
                path, map_location='cpu', weights_only=False
            )
        except Exception as error:
            rows.append({
                'checkpoint_path': str(path),
                'checkpoint_sha256': path_hash,
                'checkpoint_role': retrospective_checkpoint_role(path),
                'experiment_family': retrospective_path_family(path),
                'weight_state': '', 'state_digest': '',
                'candidate_id': '', 'architecture': 'unreadable',
                'development_fold': -1, 'fold_source': 'unresolved',
                'epoch': -1, 'metadata_parameter_count': -1,
                'state_tensor_count': 0, 'state_value_count': 0,
                'eligible_for_inference': False,
                'exclusion_reason': f'torch.load failed: {type(error).__name__}: {error}',
                'duplicate_state_aliases': 0,
            })
            continue
        if not isinstance(checkpoint, dict):
            checkpoint = {'__invalid_checkpoint__': checkpoint}
        fold, fold_source = retrospective_resolve_fold(checkpoint, path)
        epoch = checkpoint.get('epoch', checkpoint.get('best_epoch', -1))
        try:
            epoch = int(epoch)
        except (TypeError, ValueError):
            epoch = -1
        metadata_parameter_count = checkpoint.get(
            'parameter_count', checkpoint.get('parameters', -1)
        )
        try:
            metadata_parameter_count = int(metadata_parameter_count)
        except (TypeError, ValueError):
            metadata_parameter_count = -1

        observed_state = False
        for weight_state in RETROSPECTIVE_WEIGHT_STATES:
            state = checkpoint.get(weight_state)
            if not isinstance(state, dict) or not state:
                continue
            observed_state = True
            architecture, eligible, reason = retrospective_classify_state(state)
            digest = retrospective_state_digest(state)
            candidate_key = (architecture, int(fold), digest)
            candidate_id = (
                f'{architecture}_fold{int(fold)}_{digest[:16]}'
                if eligible and fold in range(5) else ''
            )
            if eligible and fold not in range(5):
                eligible = False
                reason = 'preprocessing fold could not be resolved safely'
            row_index = len(rows)
            rows.append({
                'checkpoint_path': str(path),
                'checkpoint_sha256': path_hash,
                'checkpoint_role': retrospective_checkpoint_role(path),
                'experiment_family': retrospective_path_family(path),
                'weight_state': weight_state,
                'state_digest': digest,
                'candidate_id': candidate_id,
                'architecture': architecture,
                'development_fold': int(fold),
                'fold_source': fold_source,
                'epoch': epoch,
                'metadata_parameter_count': metadata_parameter_count,
                'state_tensor_count': int(len(state)),
                'state_value_count': retrospective_state_parameter_like_count(state),
                'eligible_for_inference': bool(eligible),
                'exclusion_reason': reason,
                'duplicate_state_aliases': 0,
            })
            if eligible:
                record = unique.setdefault(candidate_key, {
                    'candidate_id': candidate_id,
                    'architecture': architecture,
                    'development_fold': int(fold),
                    'state_digest': digest,
                    'aliases': [],
                    'row_indices': [],
                })
                record['aliases'].append({
                    'checkpoint_path': str(path),
                    'checkpoint_sha256': path_hash,
                    'checkpoint_role': retrospective_checkpoint_role(path),
                    'experiment_family': retrospective_path_family(path),
                    'weight_state': weight_state,
                    'epoch': epoch,
                })
                record['row_indices'].append(row_index)
        if not observed_state:
            rows.append({
                'checkpoint_path': str(path),
                'checkpoint_sha256': path_hash,
                'checkpoint_role': retrospective_checkpoint_role(path),
                'experiment_family': retrospective_path_family(path),
                'weight_state': '', 'state_digest': '', 'candidate_id': '',
                'architecture': 'missing_state',
                'development_fold': int(fold), 'fold_source': fold_source,
                'epoch': epoch,
                'metadata_parameter_count': metadata_parameter_count,
                'state_tensor_count': 0, 'state_value_count': 0,
                'eligible_for_inference': False,
                'exclusion_reason': (
                    'neither ema_model_state nor model_state is present'
                ),
                'duplicate_state_aliases': 0,
            })
        del checkpoint
        gc.collect()

    candidate_rows = []
    for record in sorted(unique.values(), key=lambda item: item['candidate_id']):
        aliases = sorted(
            record['aliases'],
            key=lambda item: (
                0 if item['weight_state'] == 'ema_model_state' else 1,
                0 if item['checkpoint_role'] == 'best' else 1,
                item['checkpoint_path'],
            ),
        )
        representative = aliases[0]
        alias_count = len(aliases)
        for row_index in record['row_indices']:
            rows[row_index]['duplicate_state_aliases'] = alias_count
        candidate_rows.append({
            'candidate_id': record['candidate_id'],
            'architecture': record['architecture'],
            'development_fold': record['development_fold'],
            'state_digest': record['state_digest'],
            'representative_checkpoint_path': representative['checkpoint_path'],
            'representative_checkpoint_sha256': representative['checkpoint_sha256'],
            'representative_weight_state': representative['weight_state'],
            'representative_checkpoint_role': representative['checkpoint_role'],
            'representative_experiment_family': representative['experiment_family'],
            'representative_epoch': representative['epoch'],
            'alias_count': alias_count,
            'alias_records_json': json.dumps(aliases, sort_keys=True),
        })

    inventory_df = pd.DataFrame(rows).sort_values([
        'eligible_for_inference', 'experiment_family', 'checkpoint_path',
        'weight_state',
    ], ascending=[False, True, True, True]).reset_index(drop=True)
    candidate_df = pd.DataFrame(candidate_rows).sort_values(
        'candidate_id'
    ).reset_index(drop=True)
    exclusion_df = inventory_df[
        ~inventory_df['eligible_for_inference'].astype(bool)
    ].copy()
    if candidate_df.empty:
        raise RuntimeError('No safely reconstructible checkpoint states found.')
    return inventory_df, candidate_df, exclusion_df, source_records


def retrospective_sources_unchanged(source_records):
    checks = {}
    for index, record in enumerate(source_records):
        path = Path(record['path'])
        if not path.is_file():
            checks[f'source_{index:03d}_exists'] = False
            continue
        stat = path.stat()
        checks[f'source_{index:03d}_unchanged'] = bool(
            int(stat.st_size) == int(record['size_bytes'])
            and int(stat.st_mtime_ns) == int(record['mtime_ns'])
            and file_sha256(path) == record['sha256']
        )
    return checks






def retrospective_cached_local(model, architecture, batch, cache):
    if architecture == 'VascuFateNetV22':
        return pilot_local_from_cache(model, batch, cache)

    model._decoder_feature = None
    base_output = pilot_local_from_cache(model.base, batch, cache)
    if model._decoder_feature is None:
        raise RuntimeError(
            f'{architecture}: decoder feature hook did not fire.'
        )
    coarse = sample_global_to_local(
        cache['proposal_logits'], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )
    arterial_local = sample_global_to_local(
        cache['arterial_map'][:, None], batch['local_origin_in_canvas'],
        GLOBAL_CANVAS_SHAPE, LOCAL_PATCH_SIZE,
    )

    if architecture == 'VascuFateNetR8':
        satellite_logits = model.satellite_head(
            model._decoder_feature, coarse, arterial_local
        )
        confidence = torch.cat([
            batch['confidence_features'],
            torch.sigmoid(cache['routed']['lesion_presence_logit'])[:, None],
        ], dim=-1)
        gate = model.satellite_gate(confidence)
        probability = torch.sigmoid(
            base_output['logits']
            + gate[..., None, None, None] * satellite_logits
        )
        model._decoder_feature = None
        return {'probability': probability}

    if architecture == 'VascuFateNetR9':
        _, _, rescue_probability = model.rescue_head(
            batch['image'], model._decoder_feature, coarse, arterial_local
        )
        base_probability = torch.sigmoid(base_output['logits'])
        probability = torch.maximum(base_probability, rescue_probability)
        model._decoder_feature = None
        return {'probability': probability}

    raise ValueError(architecture)


@torch.no_grad()
def retrospective_infer_case(model, architecture, row, device):
    
    case = locked_load_target_free_case(row)
    base_batch = pilot_stack_sample(case['base_sample'], device)
    base_model = model if architecture == 'VascuFateNetV22' else model.base
    with torch.autocast(
        device_type=device.type, dtype=torch.bfloat16,
        enabled=device.type == 'cuda',
    ):
        cache = pilot_global_cache(base_model, base_batch)

    native_shape = case['native_shape']
    starts_by_axis = [
        pilot_sliding_starts(native_shape[axis], LOCAL_PATCH_SIZE[axis])
        for axis in range(3)
    ]
    probability_sum = np.zeros(native_shape, dtype=np.float32)
    weight_sum = np.zeros(native_shape, dtype=np.float32)
    for x in starts_by_axis[0]:
        for y in starts_by_axis[1]:
            for z in starts_by_axis[2]:
                native_start = (x, y, z)
                origin = tuple(
                    int(case['pad_start'][axis] + native_start[axis])
                    for axis in range(3)
                )
                sample = pilot_window_sample(case, origin)
                batch = pilot_stack_sample(sample, device)
                with torch.autocast(
                    device_type=device.type, dtype=torch.bfloat16,
                    enabled=device.type == 'cuda',
                ):
                    output = retrospective_cached_local(
                        model, architecture, batch, cache
                    )
                if architecture == 'VascuFateNetV22':
                    patch_probability = torch.sigmoid(
                        output['logits'].float()
                    )[0, 0].cpu().numpy()
                else:
                    patch_probability = output[
                        'probability'
                    ].float()[0, 0].cpu().numpy()
                destination = tuple(
                    slice(
                        native_start[axis],
                        min(
                            native_start[axis] + LOCAL_PATCH_SIZE[axis],
                            native_shape[axis],
                        ),
                    )
                    for axis in range(3)
                )
                copied_shape = tuple(
                    item.stop - item.start for item in destination
                )
                source = tuple(
                    slice(0, copied_shape[axis]) for axis in range(3)
                )
                weight = PILOT_WINDOW_WEIGHT[source]
                probability_sum[destination] += patch_probability[source] * weight
                weight_sum[destination] += weight
                del sample, batch, output

    if bool((weight_sum <= 0).any()):
        raise RuntimeError(
            f"{case['subject_id']}: incomplete retrospective coverage."
        )
    probability = probability_sum / weight_sum
    if not bool(
        np.isfinite(probability).all()
        and (probability >= 0).all()
        and (probability <= 1).all()
    ):
        raise RuntimeError(
            f"{case['subject_id']}: invalid local probability values."
        )

    proposal_canvas = F.interpolate(
        torch.sigmoid(cache['proposal_logits'].float()),
        size=GLOBAL_CANVAS_SHAPE, mode='trilinear', align_corners=False,
    )[0, 0].cpu().numpy()
    native_slices = tuple(
        slice(
            case['pad_start'][axis],
            case['pad_start'][axis] + native_shape[axis],
        )
        for axis in range(3)
    )
    proposal = proposal_canvas[native_slices].astype(np.float32)
    if not bool(
        np.isfinite(proposal).all()
        and (proposal >= 0).all()
        and (proposal <= 1).all()
    ):
        raise RuntimeError(
            f"{case['subject_id']}: invalid proposal probability values."
        )
    presence = float(torch.sigmoid(
        cache['routed']['lesion_presence_logit'].float()
    )[0].cpu())
    routed_volume_ml = float(torch.expm1(
        cache['routed']['log_volume_prediction'].float()
    )[0].clamp_min(0).cpu())
    local_mask = probability >= RETROSPECTIVE_LOCAL_THRESHOLD
    proposal_mask = proposal >= RETROSPECTIVE_PROPOSAL_THRESHOLD
    result = {
        'subject_id': case['subject_id'],
        'native_shape': native_shape,
        'local_mask': local_mask,
        'proposal_mask': proposal_mask,
        'local_probability_sha256': hashlib.sha256(
            probability.astype(np.float32).tobytes()
        ).hexdigest(),
        'proposal_probability_sha256': hashlib.sha256(
            proposal.astype(np.float32).tobytes()
        ).hexdigest(),
        'presence_probability': presence,
        'routed_volume_ml': routed_volume_ml,
    }
    del base_batch, cache, case, probability_sum, weight_sum
    return result


def retrospective_archive_path(candidate_id, subject_id):
    return (
        RETROSPECTIVE_PREDICTION_ROOT / candidate_id
        / f'{subject_id}_fixed_policy_masks.npz'
    )


def retrospective_validate_archive(
    path, candidate, subject_id, split, contract_hash,
):
    path = Path(path)
    if not path.is_file():
        return None
    with np.load(path, allow_pickle=False) as data:
        required = {
            'candidate_id', 'subject_id', 'dev_split', 'native_shape',
            'prediction_contract_sha256', 'checkpoint_sha256',
            'state_digest', 'architecture', 'development_fold',
            'local_threshold', 'proposal_threshold',
            'local_mask_packed', 'proposal_mask_packed',
            'local_predicted_voxels', 'proposal_predicted_voxels',
            'local_probability_sha256', 'proposal_probability_sha256',
            'presence_probability', 'routed_volume_ml',
        }
        if not required.issubset(data.files):
            raise RuntimeError(f'Incomplete retrospective archive: {path}')
        shape = tuple(int(value) for value in data['native_shape'].tolist())
        expected_bytes = (int(np.prod(shape)) + 7) // 8
        checks = {
            'candidate_matches': scalar_string(data['candidate_id'])
            == candidate['candidate_id'],
            'subject_matches': scalar_string(data['subject_id']) == subject_id,
            'split_matches': scalar_string(data['dev_split']) == split,
            'contract_matches': scalar_string(
                data['prediction_contract_sha256']
            ) == contract_hash,
            'checkpoint_matches': scalar_string(data['checkpoint_sha256'])
            == candidate['representative_checkpoint_sha256'],
            'state_matches': scalar_string(data['state_digest'])
            == candidate['state_digest'],
            'architecture_matches': scalar_string(data['architecture'])
            == candidate['architecture'],
            'fold_matches': int(np.asarray(
                data['development_fold']
            ).reshape(-1)[0]) == int(candidate['development_fold']),
            'thresholds_match': (
                math.isclose(float(np.asarray(
                    data['local_threshold']
                ).reshape(-1)[0]), RETROSPECTIVE_LOCAL_THRESHOLD)
                and math.isclose(float(np.asarray(
                    data['proposal_threshold']
                ).reshape(-1)[0]), RETROSPECTIVE_PROPOSAL_THRESHOLD)
            ),
            'shape_is_3d': len(shape) == 3,
            'local_size_matches': int(
                np.asarray(data['local_mask_packed']).size
            ) == expected_bytes,
            'proposal_size_matches': int(
                np.asarray(data['proposal_mask_packed']).size
            ) == expected_bytes,
        }
        locked_checks_or_raise(checks, f'{subject_id}: archive rejected')
    return {
        'candidate_id': candidate['candidate_id'],
        'subject_id': subject_id,
        'dev_split': split,
        'path': str(path),
        'sha256': file_sha256(path),
        'native_shape': list(shape),
    }


def retrospective_predict_candidate(
    candidate, locked_rows, contract_hash, device,
):
    candidate_id = candidate['candidate_id']
    candidate_root = RETROSPECTIVE_PREDICTION_ROOT / candidate_id
    candidate_root.mkdir(parents=True, exist_ok=True)
    candidate_gate_path = candidate_root / 'candidate_prediction_gate.json'

    if candidate_gate_path.exists():
        gate = load_json(candidate_gate_path)
        checks = {
            'prediction_complete': gate.get('prediction_complete') is True,
            'candidate_matches': gate.get('candidate_id') == candidate_id,
            'contract_matches': gate.get(
                'prediction_contract_sha256'
            ) == contract_hash,
            'subjects_exact': gate.get('subjects_predicted') == 46,
            'targets_sealed': (
                gate.get('locked_validation_targets_read') is False
                and gate.get('test_targets_read') is False
            ),
            'archives_stable': all(
                Path(record['path']).is_file()
                and file_sha256(record['path']) == record['sha256']
                for record in gate.get('prediction_manifest', [])
            ),
        }
        locked_checks_or_raise(
            checks, f'{candidate_id}: existing candidate gate rejected'
        )
        print(f'{candidate_id}: verified complete; inference skipped.')
        return gate['prediction_manifest']

    checkpoint_path = Path(candidate['representative_checkpoint_path'])
    if file_sha256(checkpoint_path) != candidate[
        'representative_checkpoint_sha256'
    ]:
        raise RuntimeError(f'{candidate_id}: checkpoint changed before load.')
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=False
    )
    weight_state = candidate['representative_weight_state']
    state = checkpoint.get(weight_state)
    if not isinstance(state, dict):
        raise RuntimeError(f'{candidate_id}: representative state disappeared.')
    if retrospective_state_digest(state) != candidate['state_digest']:
        raise RuntimeError(f'{candidate_id}: state digest changed.')

    model = retrospective_model(candidate['architecture'])
    expected_count = RETROSPECTIVE_ARCHITECTURE_COUNTS[candidate['architecture']]
    observed_count = int(sum(
        parameter.numel() for parameter in model.parameters()
    ))
    if observed_count != expected_count:
        raise RuntimeError(f'{candidate_id}: parameter count changed.')
    model.load_state_dict(state, strict=True)
    if not all(
        bool(torch.isfinite(parameter).all().item())
        for parameter in model.parameters()
    ):
        raise RuntimeError(f'{candidate_id}: non-finite model parameter.')
    model = model.to(device).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    del checkpoint, state

    global RUNTIME_FOLD
    RUNTIME_FOLD = int(candidate['development_fold'])
    manifest = []
    for index, row in enumerate(locked_rows, start=1):
        subject_id = str(row['subject_id'])
        split = str(row['dev_split'])
        archive_path = retrospective_archive_path(candidate_id, subject_id)
        existing = retrospective_validate_archive(
            archive_path, candidate, subject_id, split, contract_hash
        )
        if existing is not None:
            manifest.append(existing)
            print(
                f'{candidate_id} {index:02d}/46: {subject_id} '
                '(verified resume)'
            )
            continue
        result = retrospective_infer_case(
            model, candidate['architecture'], row, device
        )
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        locked_atomic_npz(
            archive_path,
            candidate_id=np.asarray(candidate_id),
            subject_id=np.asarray(subject_id),
            dev_split=np.asarray(split),
            native_shape=np.asarray(result['native_shape'], dtype=np.int64),
            prediction_contract_sha256=np.asarray(contract_hash),
            checkpoint_sha256=np.asarray(
                candidate['representative_checkpoint_sha256']
            ),
            state_digest=np.asarray(candidate['state_digest']),
            architecture=np.asarray(candidate['architecture']),
            development_fold=np.asarray(
                int(candidate['development_fold']), dtype=np.int64
            ),
            local_threshold=np.asarray(
                RETROSPECTIVE_LOCAL_THRESHOLD, dtype=np.float64
            ),
            proposal_threshold=np.asarray(
                RETROSPECTIVE_PROPOSAL_THRESHOLD, dtype=np.float64
            ),
            local_mask_packed=locked_pack_mask(result['local_mask']),
            proposal_mask_packed=locked_pack_mask(result['proposal_mask']),
            local_predicted_voxels=np.asarray(
                int(result['local_mask'].sum()), dtype=np.int64
            ),
            proposal_predicted_voxels=np.asarray(
                int(result['proposal_mask'].sum()), dtype=np.int64
            ),
            local_probability_sha256=np.asarray(
                result['local_probability_sha256']
            ),
            proposal_probability_sha256=np.asarray(
                result['proposal_probability_sha256']
            ),
            presence_probability=np.asarray(
                result['presence_probability'], dtype=np.float64
            ),
            routed_volume_ml=np.asarray(
                result['routed_volume_ml'], dtype=np.float64
            ),
        )
        verified = retrospective_validate_archive(
            archive_path, candidate, subject_id, split, contract_hash
        )
        manifest.append(verified)
        print(f'{candidate_id} {index:02d}/46: {subject_id}')
        del result
        gc.collect()

    gate = {
        'created_at_utc': retrospective_now(),
        'prediction_complete': True,
        'candidate_id': candidate_id,
        'architecture': candidate['architecture'],
        'development_fold': int(candidate['development_fold']),
        'state_digest': candidate['state_digest'],
        'representative_checkpoint_path': candidate[
            'representative_checkpoint_path'
        ],
        'representative_checkpoint_sha256': candidate[
            'representative_checkpoint_sha256'
        ],
        'representative_weight_state': candidate[
            'representative_weight_state'
        ],
        'prediction_contract_sha256': contract_hash,
        'subjects_predicted': 46,
        'prediction_manifest': manifest,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
    }
    retrospective_write_once_json(gate, candidate_gate_path)
    if hasattr(model, '_decoder_hook'):
        model._decoder_hook.remove()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return manifest






def retrospective_case_record(
    candidate, row, archive_path, target_record,
):
    subject_id = str(row['subject_id'])
    split = str(row['dev_split'])
    with np.load(archive_path, allow_pickle=False) as data:
        shape = tuple(int(value) for value in data['native_shape'].tolist())
        local_mask = locked_unpack_mask(data['local_mask_packed'], shape)
        proposal_mask = locked_unpack_mask(data['proposal_mask_packed'], shape)
        presence = float(np.asarray(
            data['presence_probability']
        ).reshape(-1)[0])
        routed_volume = float(np.asarray(
            data['routed_volume_ml']
        ).reshape(-1)[0])
    target = target_record['target']
    spacing = target_record['spacing']
    policy = target_record['policy']
    local = locked_mask_metrics(local_mask, target, spacing, 'local')
    proposal = locked_mask_metrics(proposal_mask, target, spacing, 'proposal')
    voxel_volume_ml = float(policy['voxel_volume_ml'])
    return {
        'candidate_id': candidate['candidate_id'],
        'candidate_kind': 'individual_saved_state',
        'architecture': candidate['architecture'],
        'experiment_family': candidate['representative_experiment_family'],
        'checkpoint_role': candidate['representative_checkpoint_role'],
        'weight_state': candidate['representative_weight_state'],
        'development_fold': int(candidate['development_fold']),
        'checkpoint_epoch': int(candidate['representative_epoch']),
        'alias_count': int(candidate['alias_count']),
        'subject_id': subject_id,
        'dev_split': split,
        'supervision_mode': policy['supervision_mode'],
        'lesion_volume_bin': policy['lesion_volume_bin'],
        'hard_metric_eligible': bool(policy['hard_metric_eligible']),
        'target_voxels': local['target_voxels'],
        'predicted_voxels': local['predicted_voxels'],
        'intersection_voxels': local['intersection_voxels'],
        'dice': local['dice'],
        'voxel_precision': local['voxel_precision'],
        'voxel_recall': local['voxel_recall'],
        'target_volume_ml': local['target_voxels'] * voxel_volume_ml,
        'predicted_volume_ml': local['predicted_voxels'] * voxel_volume_ml,
        'lesion_true_positive': local['lesion_true_positive'],
        'lesion_false_positive': local['lesion_false_positive'],
        'lesion_false_negative': local['lesion_false_negative'],
        'component_true_positive': local['component_true_positive'],
        'component_total': local['component_total'],
        'small_component_true_positive': local[
            'small_component_true_positive'
        ],
        'small_component_total': local['small_component_total'],
        'proposal_predicted_voxels': proposal['predicted_voxels'],
        'proposal_dice': proposal['dice'],
        'proposal_component_true_positive': proposal[
            'component_true_positive'
        ],
        'proposal_component_total': proposal['component_total'],
        'presence_probability': presence,
        'routed_volume_ml': routed_volume,
        'prediction_archive_path': str(archive_path),
        'prediction_archive_sha256': file_sha256(archive_path),
    }


def retrospective_official_case_rows():
    frame = pd.read_csv(LOCKED_CASE_METRICS_PATH)
    required = {
        'subject_id', 'dev_split', 'hard_metric_eligible',
        'lesion_volume_bin', 'target_voxels', 'predicted_voxels',
        'intersection_voxels', 'dice', 'target_volume_ml',
        'predicted_volume_ml', 'lesion_true_positive',
        'lesion_false_positive', 'lesion_false_negative',
        'component_true_positive', 'component_total',
        'small_component_true_positive', 'small_component_total',
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise RuntimeError(
            'Final R3 case metrics lack required columns: ' + ', '.join(missing)
        )
    frame = frame.copy()
    frame.insert(0, 'candidate_id', 'OFFICIAL_R5_FIVEFOLD_EMA_ENSEMBLE')
    frame.insert(1, 'candidate_kind', 'prospectively_frozen_official_ensemble')
    frame['architecture'] = 'VascuFateNetV22_probability_ensemble'
    frame['experiment_family'] = 'R5_final_fivefold'
    frame['checkpoint_role'] = 'strict_best_ensemble'
    frame['weight_state'] = 'ema_model_state'
    frame['development_fold'] = -1
    frame['checkpoint_epoch'] = -1
    frame['alias_count'] = 5
    return frame


def retrospective_summary_rows(case_df):
    rows = []
    identity_columns = [
        'candidate_id', 'candidate_kind', 'architecture',
        'experiment_family', 'checkpoint_role', 'weight_state',
        'development_fold', 'checkpoint_epoch', 'alias_count',
    ]
    for candidate_id, candidate_frame in case_df.groupby('candidate_id'):
        identity = {
            column: candidate_frame.iloc[0][column]
            for column in identity_columns
        }
        for split in ['val', 'test']:
            split_frame = candidate_frame[
                candidate_frame['dev_split'].astype(str) == split
            ].copy()
            if len(split_frame) != 23:
                raise RuntimeError(
                    f'{candidate_id}: {split} has {len(split_frame)} cases.'
                )
            summary = locked_aggregate_case_rows(split_frame)
            rows.append({**identity, 'dev_split': split, **summary})
    return pd.DataFrame(rows)


def retrospective_paired_bootstrap(candidate_case_df, official_case_df):
    rows = []
    official_columns = official_case_df[[
        'subject_id', 'dev_split', 'dice', 'hard_metric_eligible',
        'target_voxels', 'lesion_volume_bin',
    ]].rename(columns={'dice': 'official_dice'})
    for candidate_id, frame in candidate_case_df.groupby('candidate_id'):
        for split in ['val', 'test']:
            selected = frame[
                frame['dev_split'].astype(str) == split
            ][[
                'subject_id', 'dev_split', 'dice', 'hard_metric_eligible',
                'target_voxels', 'lesion_volume_bin',
            ]]
            merged = selected.merge(
                official_columns, on=['subject_id', 'dev_split'],
                suffixes=('', '_official'), validate='one_to_one',
            )
            eligible = merged[
                merged['hard_metric_eligible'].astype(bool)
                & (merged['target_voxels'].astype(int) > 0)
            ].copy()
            subsets = {
                'all_positive_cases': eligible,
                'tiny_small_positive_cases': eligible[
                    eligible['lesion_volume_bin'].isin(['tiny', 'small'])
                ].copy(),
            }
            for subset_name, subset in subsets.items():
                if subset.empty:
                    continue
                differences = (
                    subset['dice'].to_numpy(dtype=np.float64)
                    - subset['official_dice'].to_numpy(dtype=np.float64)
                )
                seed_offset = int(hashlib.sha256(
                    f'{candidate_id}|{split}|{subset_name}'.encode('utf-8')
                ).hexdigest()[:8], 16)
                rng = np.random.default_rng(
                    RETROSPECTIVE_SEED + seed_offset
                )
                indices = rng.integers(
                    0, len(differences),
                    size=(RETROSPECTIVE_BOOTSTRAP_REPLICATES, len(differences)),
                )
                samples = differences[indices].mean(axis=1)
                rows.append({
                    'candidate_id': candidate_id,
                    'dev_split': split,
                    'subset': subset_name,
                    'subjects': int(len(subset)),
                    'candidate_minus_official_mean_dice': float(
                        differences.mean()
                    ),
                    'ci_95_lower': float(np.quantile(samples, 0.025)),
                    'ci_95_upper': float(np.quantile(samples, 0.975)),
                    'bootstrap_standard_error': float(samples.std(ddof=1)),
                    'bootstrap_probability_difference_gt_zero': float(
                        np.mean(samples > 0)
                    ),
                    'bootstrap_replicates': RETROSPECTIVE_BOOTSTRAP_REPLICATES,
                    'method': (
                        'paired_subject_nonparametric_percentile_posthoc_'
                        'exploratory'
                    ),
                })
    return pd.DataFrame(rows)


def retrospective_rankings(summary_df):
    metrics = [
        'positive_mean_dice', 'positive_global_dice', 'lesion_f1',
        'tiny_small_subject_mean_dice', 'component_recall',
        'small_component_recall',
    ]
    rows = []
    for split in ['val', 'test']:
        frame = summary_df[
            summary_df['dev_split'].astype(str) == split
        ].copy()
        for metric in metrics:
            ranked = frame.sort_values(
                [metric, 'candidate_id'], ascending=[False, True]
            ).reset_index(drop=True)
            for rank, item in ranked.iterrows():
                rows.append({
                    'dev_split': split,
                    'metric': metric,
                    'rank': int(rank + 1),
                    'candidate_id': item['candidate_id'],
                    'candidate_kind': item['candidate_kind'],
                    'architecture': item['architecture'],
                    'experiment_family': item['experiment_family'],
                    'checkpoint_role': item['checkpoint_role'],
                    'weight_state': item['weight_state'],
                    'development_fold': int(item['development_fold']),
                    'checkpoint_epoch': int(item['checkpoint_epoch']),
                    'value': float(item[metric]),
                    'interpretation': (
                        'retrospective_exploratory_not_model_selection'
                    ),
                })
    return pd.DataFrame(rows)






def retrospective_existing_complete():
    if not RETROSPECTIVE_FINAL_GATE_PATH.exists():
        return None
    gate = load_json(RETROSPECTIVE_FINAL_GATE_PATH)
    outputs = gate.get('output_artifact_manifest', {})
    checks = {
        'audit_complete': gate.get('audit_complete') is True,
        'version_matches': gate.get('retrospective_version')
        == RETROSPECTIVE_VERSION,
        'revision_matches': gate.get('code_revision')
        == RETROSPECTIVE_CODE_REVISION,
        'scientific_status_preserved': gate.get('scientific_status')
        == 'posthoc_exploratory_nonconfirmatory',
        'official_result_not_replaced': gate.get(
            'official_result_superseded'
        ) is False,
        'artifacts_stable': bool(outputs) and all(
            Path(record['path']).is_file()
            and file_sha256(record['path']) == record['sha256']
            for record in outputs.values()
        ),
    }
    locked_checks_or_raise(
        checks,
        'Existing all-checkpoint retrospective gate failed integrity checks',
    )
    return gate


def run_trial8_all_checkpoint_retrospective():
    existing = retrospective_existing_complete()
    if existing is not None:
        print('\n' + '=' * 80)
        print('TRIAL8 ALL-CHECKPOINT RETROSPECTIVE ALREADY COMPLETE')
        print('=' * 80)
        print(
            'Unique states evaluated:             '
            f'{existing["unique_states_evaluated"]}'
        )
        print(
            'Official result superseded:          '
            f'{existing["official_result_superseded"]}'
        )
        print(f'Final gate: {RETROSPECTIVE_FINAL_GATE_PATH}')
        print('No checkpoint, prediction, or result artifact was rewritten.')
        return

    r3_gate = load_json(LOCKED_FINAL_GATE_PATH)
    r3_checks = {
        'r3_locked_evaluation_complete': r3_gate.get(
            'evaluation_complete'
        ) is True,
        'r3_targets_were_opened_after_prediction_commit': r3_gate.get(
            'targets_read_only_after_target_blind_prediction_commit'
        ) is True,
        'r3_no_post_freeze_tuning': r3_gate.get(
            'post_freeze_tuning_performed'
        ) is False,
        'r3_local_threshold_frozen': math.isclose(
            float(r3_gate.get('frozen_local_threshold', float('nan'))),
            RETROSPECTIVE_LOCAL_THRESHOLD,
        ),
        'r3_proposal_threshold_frozen': math.isclose(
            float(r3_gate.get('frozen_proposal_threshold', float('nan'))),
            RETROSPECTIVE_PROPOSAL_THRESHOLD,
        ),
        'r3_case_metrics_exist': Path(LOCKED_CASE_METRICS_PATH).is_file(),
    }
    locked_checks_or_raise(r3_checks, 'R3 locked result prerequisite failed')

    manifest = pd.read_csv(MULTIMODAL_MANIFEST_PATH)
    manifest['subject_id'] = manifest['subject_id'].astype(str)
    locked_frame = manifest[
        manifest['dev_split'].astype(str).isin(['val', 'test'])
    ].copy().sort_values(['dev_split', 'subject_id'])
    split_counts = locked_frame['dev_split'].value_counts().to_dict()
    manifest_checks = {
        'locked_subjects_exact': len(locked_frame) == 46,
        'locked_subjects_unique': locked_frame['subject_id'].is_unique,
        'locked_split_counts_exact': split_counts
        == RETROSPECTIVE_EXPECTED_SPLITS,
        **locked_manifest_input_contract_checks(locked_frame),
    }
    locked_checks_or_raise(
        manifest_checks, 'Retrospective locked-input contract failed'
    )

    print('\n' + '=' * 80)
    print('TRIAL8 ALL-CHECKPOINT RETROSPECTIVE INVENTORY STARTING')
    print('=' * 80)
    print('Scientific status:                  post-hoc exploratory')
    print('Official R5 result remains primary: True')
    print('Local / proposal threshold:         0.40 / 0.35')
    print('Postprocessing / TTA:               none / none')
    print('Locked targets read during phase 1: False')
    inventory_df, candidate_df, exclusion_df, source_records = (
        retrospective_inventory()
    )

    contract = {
        'retrospective_version': RETROSPECTIVE_VERSION,
        'code_revision': RETROSPECTIVE_CODE_REVISION,
        'scientific_status': 'posthoc_exploratory_nonconfirmatory',
        'official_result_superseded': False,
        'local_threshold': RETROSPECTIVE_LOCAL_THRESHOLD,
        'proposal_threshold': RETROSPECTIVE_PROPOSAL_THRESHOLD,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'weight_states_inventoried': list(RETROSPECTIVE_WEIGHT_STATES),
        'state_deduplication_key': [
            'architecture', 'development_fold', 'state_digest'
        ],
        'multimodal_manifest_sha256': file_sha256(
            MULTIMODAL_MANIFEST_PATH
        ),
        'multimodal_gate_sha256': file_sha256(MULTIMODAL_GATE_PATH),
        'r3_final_gate_sha256': file_sha256(LOCKED_FINAL_GATE_PATH),
        'r3_case_metrics_sha256': file_sha256(LOCKED_CASE_METRICS_PATH),
        'subject_ids': locked_frame['subject_id'].tolist(),
        'candidate_records': candidate_df.to_dict('records'),
        'checkpoint_source_records': source_records,
        'target_access_before_prediction_commit': False,
    }
    contract_hash = locked_canonical_hash(contract)
    contract['prediction_contract_sha256'] = contract_hash

    RETROSPECTIVE_ROOT.mkdir(parents=True, exist_ok=True)
    retrospective_write_once_csv(inventory_df, RETROSPECTIVE_INVENTORY_PATH)
    retrospective_write_once_csv(candidate_df, RETROSPECTIVE_CANDIDATE_PATH)
    retrospective_write_once_csv(exclusion_df, RETROSPECTIVE_EXCLUSION_PATH)
    retrospective_write_once_json(contract, RETROSPECTIVE_CONTRACT_PATH)
    source_checks = retrospective_sources_unchanged(source_records)
    locked_checks_or_raise(
        source_checks, 'Checkpoint sources changed after inventory'
    )

    if not torch.cuda.is_available():
        raise RuntimeError(
            'The target-blind retrospective inference phase requires CUDA.'
        )
    device = torch.device('cuda')
    set_determinism(RETROSPECTIVE_SEED)
    gc.collect()
    torch.cuda.empty_cache()
    prediction_manifest = []
    locked_rows = locked_frame.to_dict('records')
    print('\n' + '=' * 80)
    print('TRIAL8 TARGET-BLIND ALL-CHECKPOINT INFERENCE STARTING')
    print('=' * 80)
    print(f'Unique reconstructible states:      {len(candidate_df)}')
    print('Locked validation targets read:     False')
    print('Test targets read:                  False')
    print('Optimizer constructed / steps:      False / 0')
    for index, candidate in enumerate(
        candidate_df.to_dict('records'), start=1
    ):
        print('\n' + '-' * 80)
        print(
            f'Candidate {index}/{len(candidate_df)}: '
            f'{candidate["candidate_id"]}'
        )
        manifest_records = retrospective_predict_candidate(
            candidate, locked_rows, contract_hash, device
        )
        prediction_manifest.extend(manifest_records)

    if len(prediction_manifest) != 46 * len(candidate_df):
        raise RuntimeError('Target-blind prediction manifest is incomplete.')
    locked_checks_or_raise(
        retrospective_sources_unchanged(source_records),
        'Checkpoint sources changed during target-blind inference',
    )
    prediction_gate = {
        'prediction_complete': True,
        'retrospective_version': RETROSPECTIVE_VERSION,
        'code_revision': RETROSPECTIVE_CODE_REVISION,
        'prediction_contract_sha256': contract_hash,
        'unique_states_predicted': int(len(candidate_df)),
        'subjects_per_state': 46,
        'prediction_archives': int(len(prediction_manifest)),
        'prediction_manifest': prediction_manifest,
        'locked_validation_targets_read': False,
        'test_targets_read': False,
        'targets_may_open_only_after_this_gate': True,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
    }
    retrospective_write_once_json(
        prediction_gate, RETROSPECTIVE_PREDICTION_GATE_PATH
    )
    print('\nTarget-blind prediction gate committed.')
    print('Loading both locked target sets for retrospective scoring only.')
    target_cache = {}
    target_policy_rows = []
    for row in locked_rows:
        subject_id = str(row['subject_id'])
        first_candidate = candidate_df.iloc[0].to_dict()
        first_archive = retrospective_archive_path(
            first_candidate['candidate_id'], subject_id
        )
        with np.load(first_archive, allow_pickle=False) as data:
            shape = tuple(int(value) for value in data['native_shape'].tolist())
        target, soft, spacing = locked_load_target(row, shape)
        policy = locked_derive_target_policy(target, soft, spacing)
        target_cache[subject_id] = {
            'target': target, 'soft': soft, 'spacing': spacing,
            'policy': policy,
        }
        target_policy_rows.append({
            'subject_id': subject_id,
            'dev_split': str(row['dev_split']),
            'supervision_mode': policy['supervision_mode'],
            'hard_metric_eligible': policy['hard_metric_eligible'],
            'lesion_volume_bin': policy['lesion_volume_bin'],
            'hard_voxels': policy['hard_voxels'],
            'soft_support_voxels': policy['soft_support_voxels'],
            'effective_target_volume_ml': policy[
                'effective_target_volume_ml'
            ],
            'policy_source': 'target_derived_after_prediction_commit',
        })
    if len(target_cache) != 46:
        raise RuntimeError('Both locked target sets were not loaded completely.')
    target_policy_df = pd.DataFrame(target_policy_rows).sort_values(
        ['dev_split', 'subject_id']
    )

    case_rows = []
    for candidate in candidate_df.to_dict('records'):
        for row in locked_rows:
            subject_id = str(row['subject_id'])
            archive_path = retrospective_archive_path(
                candidate['candidate_id'], subject_id
            )
            case_rows.append(retrospective_case_record(
                candidate, row, archive_path, target_cache[subject_id]
            ))
    candidate_case_df = pd.DataFrame(case_rows)
    official_case_df = retrospective_official_case_rows()
    combined_case_df = pd.concat(
        [candidate_case_df, official_case_df],
        ignore_index=True, sort=False,
    )
    summary_df = retrospective_summary_rows(combined_case_df)
    ranking_df = retrospective_rankings(summary_df)
    bootstrap_df = retrospective_paired_bootstrap(
        candidate_case_df, official_case_df
    )

    family_rows = []
    individual_summary = summary_df[
        summary_df['candidate_kind'] == 'individual_saved_state'
    ].copy()
    for family, family_frame in individual_summary.groupby('experiment_family'):
        validation = family_frame[
            family_frame['dev_split'] == 'val'
        ].sort_values(
            ['positive_mean_dice', 'candidate_id'],
            ascending=[False, True],
        )
        if validation.empty:
            continue
        best = validation.iloc[0]
        test = family_frame[
            (family_frame['dev_split'] == 'test')
            & (family_frame['candidate_id'] == best['candidate_id'])
        ].iloc[0]
        family_rows.append({
            'experiment_family': family,
            'unique_states': int(family_frame['candidate_id'].nunique()),
            'validation_selected_state': best['candidate_id'],
            'validation_positive_mean_dice': float(
                best['positive_mean_dice']
            ),
            'test_positive_mean_dice_of_validation_selected_state': float(
                test['positive_mean_dice']
            ),
            'test_positive_global_dice_of_validation_selected_state': float(
                test['positive_global_dice']
            ),
            'selection_status': (
                'posthoc_exploratory_validation_ranking_only'
            ),
        })
    family_df = pd.DataFrame(family_rows).sort_values(
        'validation_positive_mean_dice', ascending=False
    )

    val_individual = individual_summary[
        individual_summary['dev_split'] == 'val'
    ].sort_values(
        ['positive_mean_dice', 'candidate_id'], ascending=[False, True]
    )
    test_individual = individual_summary[
        individual_summary['dev_split'] == 'test'
    ].sort_values(
        ['positive_mean_dice', 'candidate_id'], ascending=[False, True]
    )
    official_summary = summary_df[
        summary_df['candidate_id']
        == 'OFFICIAL_R5_FIVEFOLD_EMA_ENSEMBLE'
    ].set_index('dev_split')
    best_val = val_individual.iloc[0]
    best_test = test_individual.iloc[0]

    final_checks = {
        **{f'manifest_{name}': bool(value)
           for name, value in manifest_checks.items()},
        'inventory_has_checkpoint_files': len(inventory_df) > 0,
        'unique_candidates_nonzero': len(candidate_df) > 0,
        'all_candidates_have_46_cases': bool(
            candidate_case_df.groupby('candidate_id').size().eq(46).all()
        ),
        'all_candidates_have_both_splits': bool(
            candidate_case_df.groupby(
                ['candidate_id', 'dev_split']
            ).size().eq(23).all()
        ),
        'candidate_case_metrics_finite': bool(np.isfinite(
            candidate_case_df[[
                'dice', 'voxel_precision', 'voxel_recall',
                'target_volume_ml', 'predicted_volume_ml',
            ]].to_numpy(dtype=float)
        ).all()),
        'summary_rows_complete': len(summary_df)
        == 2 * (len(candidate_df) + 1),
        'paired_bootstrap_complete': len(bootstrap_df)
        == 4 * len(candidate_df),
        'fixed_local_threshold_unchanged': (
            RETROSPECTIVE_LOCAL_THRESHOLD == 0.40
        ),
        'fixed_proposal_threshold_unchanged': (
            RETROSPECTIVE_PROPOSAL_THRESHOLD == 0.35
        ),
        'postprocessing_none': True,
        'tta_none': True,
        'official_ensemble_retained_as_primary': True,
        'retrospective_result_not_confirmatory': True,
        'checkpoint_sources_unchanged': all(
            retrospective_sources_unchanged(source_records).values()
        ),
        'optimizer_not_constructed': True,
        'optimizer_steps_are_zero': True,
        'training_not_performed': True,
        'posthoc_tuning_not_authorized': True,
    }
    locked_checks_or_raise(final_checks, 'Retrospective final audit failed')

    retrospective_write_once_csv(
        target_policy_df, RETROSPECTIVE_TARGET_POLICY_PATH
    )
    retrospective_write_once_csv(
        combined_case_df, RETROSPECTIVE_CASE_METRICS_PATH
    )
    retrospective_write_once_csv(
        summary_df, RETROSPECTIVE_SPLIT_SUMMARY_PATH
    )
    retrospective_write_once_csv(
        ranking_df, RETROSPECTIVE_RANKING_PATH
    )
    retrospective_write_once_csv(
        bootstrap_df, RETROSPECTIVE_BOOTSTRAP_PATH
    )
    retrospective_write_once_csv(family_df, RETROSPECTIVE_FAMILY_PATH)

    report = {
        'retrospective_version': RETROSPECTIVE_VERSION,
        'code_revision': RETROSPECTIVE_CODE_REVISION,
        'scientific_status': 'posthoc_exploratory_nonconfirmatory',
        'official_result_superseded': False,
        'official_primary_result': {
            split: {
                'positive_mean_dice': float(
                    official_summary.loc[split, 'positive_mean_dice']
                ),
                'positive_global_dice': float(
                    official_summary.loc[split, 'positive_global_dice']
                ),
                'lesion_f1': float(
                    official_summary.loc[split, 'lesion_f1']
                ),
            }
            for split in ['val', 'test']
        },
        'best_individual_state_by_validation_mean_dice': {
            'candidate_id': best_val['candidate_id'],
            'validation_positive_mean_dice': float(
                best_val['positive_mean_dice']
            ),
            'test_positive_mean_dice': float(
                individual_summary.loc[
                    (individual_summary['candidate_id']
                     == best_val['candidate_id'])
                    & (individual_summary['dev_split'] == 'test'),
                    'positive_mean_dice',
                ].iloc[0]
            ),
            'interpretation': (
                'exploratory_validation_selected_after_locked_results_opened'
            ),
        },
        'best_individual_state_by_test_mean_dice': {
            'candidate_id': best_test['candidate_id'],
            'test_positive_mean_dice': float(
                best_test['positive_mean_dice']
            ),
            'interpretation': (
                'descriptive_test_winner_not_valid_for_model_selection'
            ),
        },
        'checkpoint_files_inventoried': int(len(source_records)),
        'state_records_inventoried': int(len(inventory_df)),
        'unique_states_evaluated': int(len(candidate_df)),
        'excluded_state_records': int(len(exclusion_df)),
        'frozen_local_threshold': RETROSPECTIVE_LOCAL_THRESHOLD,
        'frozen_proposal_threshold': RETROSPECTIVE_PROPOSAL_THRESHOLD,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'predictions_committed_before_targets': True,
        'model_training_performed': False,
        'posthoc_tuning_authorized': False,
        'reporting_rule': (
            'Official R5 five-fold EMA ensemble remains the thesis result. '
            'All checkpoint rankings are labeled retrospective exploratory '
            'and must not be presented as an unbiased model-selection result.'
        ),
    }
    retrospective_write_once_json(report, RETROSPECTIVE_REPORT_PATH)

    output_paths = {
        'checkpoint_state_inventory': RETROSPECTIVE_INVENTORY_PATH,
        'unique_inference_candidates': RETROSPECTIVE_CANDIDATE_PATH,
        'excluded_checkpoint_states': RETROSPECTIVE_EXCLUSION_PATH,
        'prediction_contract': RETROSPECTIVE_CONTRACT_PATH,
        'target_blind_prediction_gate': RETROSPECTIVE_PREDICTION_GATE_PATH,
        'target_derived_policy_audit': RETROSPECTIVE_TARGET_POLICY_PATH,
        'checkpoint_case_metrics': RETROSPECTIVE_CASE_METRICS_PATH,
        'checkpoint_split_summary': RETROSPECTIVE_SPLIT_SUMMARY_PATH,
        'exploratory_rankings': RETROSPECTIVE_RANKING_PATH,
        'paired_bootstrap_vs_official': RETROSPECTIVE_BOOTSTRAP_PATH,
        'architecture_family_summary': RETROSPECTIVE_FAMILY_PATH,
        'retrospective_report': RETROSPECTIVE_REPORT_PATH,
    }
    output_manifest = {
        name: {'path': str(path), 'sha256': file_sha256(path)}
        for name, path in output_paths.items()
    }
    gate = {
        'created_at_utc': retrospective_now(),
        'audit_complete': True,
        'retrospective_version': RETROSPECTIVE_VERSION,
        'code_revision': RETROSPECTIVE_CODE_REVISION,
        'scientific_status': 'posthoc_exploratory_nonconfirmatory',
        'official_result_superseded': False,
        'unique_states_evaluated': int(len(candidate_df)),
        'checkpoint_files_inventoried': int(len(source_records)),
        'excluded_state_records': int(len(exclusion_df)),
        'evaluated_splits': ['val', 'test'],
        'subjects_per_split': RETROSPECTIVE_EXPECTED_SPLITS,
        'frozen_local_threshold': RETROSPECTIVE_LOCAL_THRESHOLD,
        'frozen_proposal_threshold': RETROSPECTIVE_PROPOSAL_THRESHOLD,
        'postprocessing': 'none',
        'test_time_augmentation': 'none',
        'prediction_contract_sha256': contract_hash,
        'predictions_committed_before_targets': True,
        'locked_validation_targets_read_after_prediction_gate': True,
        'test_targets_read_after_prediction_gate': True,
        'optimizer_constructed': False,
        'optimizer_steps': 0,
        'training_performed': False,
        'posthoc_tuning_authorized': False,
        'final_checks': final_checks,
        'output_artifact_manifest': output_manifest,
    }
    retrospective_write_once_json(gate, RETROSPECTIVE_FINAL_GATE_PATH)

    print('\nCheckpoint-state inventory summary:')
    display(inventory_df.groupby([
        'experiment_family', 'architecture', 'eligible_for_inference'
    ], dropna=False).size().reset_index(name='state_records'))
    print('\nTop validation mean-Dice rows (exploratory):')
    display(summary_df[
        summary_df['dev_split'] == 'val'
    ].sort_values('positive_mean_dice', ascending=False)[[
        'candidate_id', 'candidate_kind', 'experiment_family',
        'weight_state', 'positive_mean_dice', 'positive_global_dice',
        'lesion_f1', 'tiny_small_subject_mean_dice',
    ]].head(15))
    print('\nCorresponding test rows (descriptive only):')
    validation_order = summary_df[
        summary_df['dev_split'] == 'val'
    ].sort_values('positive_mean_dice', ascending=False)[
        'candidate_id'
    ].tolist()
    test_display = summary_df[
        summary_df['dev_split'] == 'test'
    ].copy()
    test_display['validation_order'] = test_display['candidate_id'].map(
        {candidate_id: index for index, candidate_id in enumerate(validation_order)}
    )
    display(test_display.sort_values('validation_order')[[
        'candidate_id', 'candidate_kind', 'experiment_family',
        'weight_state', 'positive_mean_dice', 'positive_global_dice',
        'lesion_f1', 'tiny_small_subject_mean_dice',
    ]].head(15))

    print('\n' + '=' * 80)
    print('TRIAL8 ALL-CHECKPOINT RETROSPECTIVE AUDIT PASSED')
    print('=' * 80)
    print(f'Checkpoint files inventoried:        {len(source_records)}')
    print(f'Unique states evaluated:             {len(candidate_df)}')
    print(f'Excluded/unsupported state records:  {len(exclusion_df)}')
    print('Scientific status:                   post-hoc exploratory')
    print('Official R5 ensemble superseded:     False')
    print('Thresholds / TTA / postprocessing:   0.40, 0.35 / none / none')
    print('Predictions committed before targets: True')
    print('Training / optimizer steps:          False / 0')
    print(
        'Best individual by validation mean Dice: '
        f'{best_val["candidate_id"]} ({best_val["positive_mean_dice"]:.4f})'
    )
    print(
        'Best individual by test mean Dice:       '
        f'{best_test["candidate_id"]} ({best_test["positive_mean_dice"]:.4f}) '
        '[descriptive only]'
    )
    print(f'Report:                              {RETROSPECTIVE_REPORT_PATH}')
    print(f'Final gate:                          {RETROSPECTIVE_FINAL_GATE_PATH}')
    print(
        '\nUse the official frozen R5 five-fold EMA ensemble for thesis headline '
        'results. Use this audit only as explicitly labeled post-hoc analysis.'
    )


run_trial8_all_checkpoint_retrospective()


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import ndimage

ROOT = Path("/path/to/project")
MODEL = "trial8_vascufate_net_v2_2_global_local_arterial"

MANIFEST = (
    ROOT / "metadata"
    / "model_manifest_v2_vascufate_r2_multimodal_loader_candidate.csv"
)
R3 = (
    ROOT / "evaluation" / MODEL
    / "v3_final_locked_val_test_metric_reconciliation"
)
R2 = (
    ROOT / "evaluation" / MODEL
    / "v2_final_locked_val_test_fivefold_ensemble_once"
)

FINAL_GATE = R3 / "locked_evaluation_gate.json"
FINAL_REPORT = R3 / "locked_evaluation_report.json"
CASE_CSV = R3 / "locked_case_metrics.csv"
COMPONENT_CSV = R3 / "locked_component_metrics.csv"
PREDICTION_GATE = R2 / "target_blind_prediction_gate.json"
PREDICTION_DIR = R2 / "target_blind_predictions"


OUT = (
    ROOT / "thesis_figures"
    / "trial8_final_validation_verified_masks_v1"
)

EXPECTED_MANIFEST_SHA256 = (
    "6c9301cdeffffb9c4f1cdbc17eace05a865d35d87432948e8ae629c5e7794235"
)
FROZEN_LOCAL_THRESHOLD = 0.40
FROZEN_PROPOSAL_THRESHOLD = 0.35
STRUCTURE_26 = np.ones((3, 3, 3), dtype=np.uint8)


def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path):
    with open(path, "r", encoding="utf-8") as stream:
        return json.load(stream)


def scalar_text(array):
    value = np.asarray(array).reshape(-1)[0]
    return value.decode() if isinstance(value, bytes) else str(value)


def unpack_mask(packed, shape):
    count = int(np.prod(shape))
    bits = np.asarray(packed, dtype=np.uint8)
    expected_bytes = (count + 7) // 8
    require(bits.size == expected_bytes, "Packed mask has wrong byte count.")
    return np.unpackbits(
        bits, bitorder="little"
    )[:count].reshape(shape).astype(bool)


def source_path_in_fresh_mount(stored_path):
    
    stored = Path(str(stored_path))
    old_root = Path("/path/to/project")

    if stored.is_relative_to(old_root):
        return ROOT / stored.relative_to(old_root)
    if stored.is_relative_to(ROOT):
        return stored

    raise RuntimeError(
        f"Manifest NPZ path is outside the verified project: {stored}"
    )


def image_slice(volume, z):
    return np.rot90(np.asarray(volume)[..., z])


def display_window(slice_2d):
    image = np.asarray(slice_2d, dtype=np.float32)
    finite = image[np.isfinite(image)]
    require(finite.size > 0, "Image slice contains no finite pixels.")
    lo, hi = np.percentile(finite, [1, 99])
    if hi <= lo:
        hi = lo + 1.0
    return np.clip((image - lo) / (hi - lo), 0, 1)


def best_target_slice(target):
    
    counts = np.asarray(target, dtype=bool).sum(axis=(0, 1))
    return int(np.argmax(counts))


def overlay(ax, base, target=None, prediction=None):
    ax.imshow(display_window(base), cmap="gray", interpolation="nearest")

    if target is not None and np.any(target):
        green = np.zeros((*target.shape, 4), dtype=np.float32)
        green[..., 1] = 1
        green[..., 3] = np.asarray(target, bool) * 0.45
        ax.imshow(green, interpolation="nearest")

    if prediction is not None and np.any(prediction):
        red = np.zeros((*prediction.shape, 4), dtype=np.float32)
        red[..., 0] = 1
        red[..., 3] = np.asarray(prediction, bool) * 0.40
        ax.imshow(red, interpolation="nearest")

    ax.axis("off")






required_files = [
    MANIFEST, FINAL_GATE, FINAL_REPORT, CASE_CSV, COMPONENT_CSV,
    PREDICTION_GATE,
]
missing = [str(path) for path in required_files if not path.is_file()]
require(not missing, "Missing frozen source artifact(s):\n" + "\n".join(missing))
require(PREDICTION_DIR.is_dir(), f"Missing prediction directory: {PREDICTION_DIR}")

require(
    sha256_file(MANIFEST) == EXPECTED_MANIFEST_SHA256,
    "Multimodal manifest hash does not match frozen Trial8 manifest."
)

final_gate = read_json(FINAL_GATE)
report = read_json(FINAL_REPORT)
prediction_gate = read_json(PREDICTION_GATE)

require(final_gate.get("evaluation_complete") is True,
        "Final R3 evaluation gate is not complete.")
require(final_gate.get("evaluation_version")
        == "v3_final_locked_val_test_metric_reconciliation",
        "Unexpected final evaluation version.")
require(final_gate.get("subjects_evaluated") == {"val": 23, "test": 23},
        "Final evaluation cohort sizes changed.")
require(final_gate.get("frozen_local_threshold") == FROZEN_LOCAL_THRESHOLD,
        "Final local threshold changed.")
require(final_gate.get("frozen_proposal_threshold")
        == FROZEN_PROPOSAL_THRESHOLD,
        "Final proposal threshold changed.")
require(final_gate.get("ensemble_members") == 5,
        "Final ensemble is not the five-model ensemble.")
require(final_gate.get("postprocessing") == "none"
        and final_gate.get("test_time_augmentation") == "none",
        "Final inference policy changed.")
require(
    final_gate.get("final_checks")
    and all(final_gate["final_checks"].values()),
    "One or more final R3 checks did not pass."
)

require(report.get("frozen_local_threshold") == FROZEN_LOCAL_THRESHOLD,
        "R3 report disagrees on local threshold.")
require(report.get("frozen_proposal_threshold")
        == FROZEN_PROPOSAL_THRESHOLD,
        "R3 report disagrees on proposal threshold.")

require(prediction_gate.get("prediction_complete") is True,
        "Target-blind prediction gate is incomplete.")
require(prediction_gate.get("subjects_predicted") == 46,
        "Target-blind prediction cohort size changed.")
require(prediction_gate.get("ensemble_members") == 5,
        "Target-blind prediction was not a five-model ensemble.")
require(prediction_gate.get("locked_validation_targets_read") is False
        and prediction_gate.get("test_targets_read") is False,
        "Target-blind prediction provenance check failed.")
require(
    prediction_gate.get("checks")
    and all(prediction_gate["checks"].values()),
    "One or more target-blind prediction checks did not pass."
)


outputs = final_gate.get("output_artifact_manifest", {})
for key, path in {
    "target_blind_prediction_gate": PREDICTION_GATE,
    "case_metrics": CASE_CSV,
    "component_metrics": COMPONENT_CSV,
    "evaluation_report": FINAL_REPORT,
}.items():
    record = outputs.get(key, {})
    require(
        record.get("sha256") == sha256_file(path),
        f"Final R3 gate hash disagrees with {key}."
    )

manifest = pd.read_csv(MANIFEST)
case_df = pd.read_csv(CASE_CSV)
component_df = pd.read_csv(COMPONENT_CSV)

for column in [
    "subject_id", "dev_split", "vascufate_v2_npz_path",
    "trial8_npz_sha256"
]:
    require(column in manifest.columns, f"Missing manifest column: {column}")

require(
    set(manifest["dev_split"].astype(str)) == {"train", "val", "test"},
    "Unexpected manifest split names."
)
require(
    manifest["dev_split"].value_counts().to_dict()
    == {"train": 102, "val": 23, "test": 23},
    "Manifest split counts changed."
)

val_manifest = manifest[
    manifest["dev_split"].astype(str) == "val"
].copy().set_index("subject_id", drop=False)

val_cases = case_df[
    case_df["dev_split"].astype(str) == "val"
].copy().set_index("subject_id", drop=False)

val_components = component_df[
    component_df["dev_split"].astype(str) == "val"
].copy()

require(len(val_manifest) == len(val_cases) == 23,
        "Validation manifest/metric row counts are not 23.")
require(
    set(val_manifest.index) == set(val_cases.index),
    "Validation subjects disagree between manifest and reconciled metrics."
)
require(
    np.allclose(
        val_cases["local_threshold"].to_numpy(dtype=float),
        FROZEN_LOCAL_THRESHOLD
    ),
    "Validation case metrics use another local threshold."
)

archive_index = {
    str(item["subject_id"]): item
    for item in prediction_gate["prediction_manifest"]
}
require(len(archive_index) == 46,
        "Target-blind prediction manifest is incomplete.")

contract_sha = prediction_gate["prediction_contract_sha256"]


def archive_path_for(subject_id):
    return (
        PREDICTION_DIR
        / f"{subject_id}_fivefold_ensemble_masks.npz"
    )




for sid in val_manifest.index:
    require(sid in archive_index, f"{sid}: absent from prediction gate.")
    record = archive_index[sid]
    archive = archive_path_for(sid)
    require(archive.is_file(), f"{sid}: prediction archive missing.")
    require(sha256_file(archive) == record["sha256"],
            f"{sid}: prediction archive SHA-256 mismatch.")

    with np.load(archive, allow_pickle=False) as data:
        required_keys = {
            "subject_id", "dev_split", "native_shape", "contract_sha256",
            "fold_ids", "fold_local_masks_packed",
            "ensemble_local_mask_packed",
            "ensemble_proposal_mask_packed",
        }
        require(required_keys.issubset(data.files),
                f"{sid}: archive is missing frozen mask fields.")
        require(scalar_text(data["subject_id"]) == sid,
                f"{sid}: archive subject ID mismatch.")
        require(scalar_text(data["dev_split"]) == "val",
                f"{sid}: archive is not marked validation.")
        require(scalar_text(data["contract_sha256"]) == contract_sha,
                f"{sid}: prediction contract mismatch.")
        require(
            tuple(np.asarray(data["fold_ids"], dtype=int)) == (0, 1, 2, 3, 4),
            f"{sid}: unexpected member IDs."
        )

print("Frozen R3 gate, metrics, manifest, and all 23 validation archive hashes verified.")
print("Validation targets have not yet been opened; test targets will not be opened.")


def load_case(subject_id):
    
    require(subject_id in val_manifest.index,
            f"Refusing to open a non-validation case: {subject_id}")

    row = val_manifest.loc[subject_id]
    npz_path = source_path_in_fresh_mount(row["vascufate_v2_npz_path"])

    require(npz_path.is_file(), f"{subject_id}: missing input NPZ: {npz_path}")
    require(
        sha256_file(npz_path) == row["trial8_npz_sha256"],
        f"{subject_id}: input NPZ hash differs from frozen manifest."
    )

    archive_path = archive_path_for(subject_id)

    with np.load(archive_path, allow_pickle=False) as data:
        shape = tuple(int(x) for x in data["native_shape"])
        ensemble = unpack_mask(
            data["ensemble_local_mask_packed"], shape
        )
        members_packed = np.asarray(
            data["fold_local_masks_packed"], dtype=np.uint8
        )
        require(members_packed.shape[0] == 5,
                f"{subject_id}: missing member masks.")
        member0 = unpack_mask(members_packed[0], shape)

    with np.load(npz_path, allow_pickle=False) as data:
        require(
            {"image", "input_channels", "target_binary",
             "target_soft", "target_spacing"}.issubset(data.files),
            f"{subject_id}: input/target schema incomplete."
        )
        channels = [scalar_text(x) for x in data["input_channels"]]
        require("ncct_brain_window" in channels,
                f"{subject_id}: expected NCCT channel unavailable.")

        channel_index = channels.index("ncct_brain_window")
        image = np.asarray(
            data["image"][channel_index], dtype=np.float32
        )
        target = np.asarray(data["target_binary"]) > 0
        spacing = np.asarray(data["target_spacing"], dtype=float)

    require(image.shape == target.shape == ensemble.shape,
            f"{subject_id}: image/target/prediction shape mismatch.")
    require(spacing.shape == (3,) and np.all(spacing > 0),
            f"{subject_id}: invalid target spacing.")

    official = val_cases.loc[subject_id]
    require(int(target.sum()) == int(official["target_voxels"]),
            f"{subject_id}: target disagrees with final R3 metrics.")
    require(int(ensemble.sum()) == int(official["predicted_voxels"]),
            f"{subject_id}: ensemble disagrees with final R3 metrics.")

    intersection = int(np.logical_and(target, ensemble).sum())
    require(intersection == int(official["intersection_voxels"]),
            f"{subject_id}: overlap disagrees with final R3 metrics.")

    return {
        "id": subject_id,
        "image": image,
        "target": target,
        "ensemble": ensemble,
        "member0": member0,
        "archive": archive_path,
        "npz": npz_path,
        "official": official,
    }






positive = val_cases[
    val_cases["target_voxels"].astype(int) > 0
].copy().reset_index(drop=True).sort_values(
    ["target_volume_ml", "subject_id"]
)

require(len(positive) >= 3, "Not enough positive validation cases.")


main_id = str(positive.iloc[len(positive) // 2]["subject_id"])

scale_ids = [
    str(positive.iloc[round(q * (len(positive) - 1))]["subject_id"])
    for q in (0.10, 0.50, 0.90)
]

component_counts = (
    val_components.groupby("subject_id").size()
    .sort_values(ascending=False)
)
require(not component_counts.empty, "No validation component records found.")
component_id = str(component_counts.index[0])

raw_ids = [
    str(x) for x in val_manifest.index.sort_values()[:3]
]
required_ids = list(dict.fromkeys(
    raw_ids + [main_id] + scale_ids + [component_id]
))


cases = {sid: load_case(sid) for sid in required_ids}

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Output directory already contains files: {OUT}\n"
        "Existing figures were not overwritten."
    )

OUT.mkdir(parents=True, exist_ok=True)

legend = [
    Patch(facecolor="lime", alpha=0.6, label="Final infarct reference"),
    Patch(facecolor="red", alpha=0.6, label="Frozen prediction"),
]






fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, sid in zip(axes, raw_ids):
    case = cases[sid]
    z = best_target_slice(case["target"])
    ax.imshow(
        display_window(image_slice(case["image"], z)),
        cmap="gray"
    )
    ax.set_title(f"{sid} | pretreatment NCCT")
    ax.axis("off")

fig.suptitle("Representative pretreatment NCCT inputs")
fig.tight_layout()
fig.savefig(OUT / "01_pretreatment_ncct_samples.png",
            dpi=300, bbox_inches="tight")
plt.close(fig)






case = cases[main_id]
z = best_target_slice(case["target"])
base = image_slice(case["image"], z)
reference = image_slice(case["target"], z)
predicted = image_slice(case["ensemble"], z)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(display_window(base), cmap="gray")
axes[0].set_title("Pretreatment NCCT")

axes[1].imshow(reference, cmap="gray")
axes[1].set_title("Final infarct reference")

axes[2].imshow(predicted, cmap="gray")
axes[2].set_title("Frozen ensemble mask")

overlay(axes[3], base, reference, predicted)
axes[3].set_title("Reference / prediction")
axes[3].legend(handles=legend, loc="lower right", fontsize=7)

for ax in axes:
    ax.axis("off")

fig.suptitle(
    f"{main_id} | locked validation | "
    f"Dice = {float(case['official']['dice']):.3f}"
)
fig.tight_layout()
fig.savefig(OUT / "02_ncct_reference_ensemble_overlay.png",
            dpi=300, bbox_inches="tight")
plt.close(fig)






member = image_slice(case["member0"], z)

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))

axes[0].imshow(display_window(base), cmap="gray")
axes[0].set_title("Pretreatment NCCT")

axes[1].imshow(reference, cmap="gray")
axes[1].set_title("Final infarct reference")

axes[2].imshow(member, cmap="gray")
axes[2].set_title("Frozen fold-0 member mask")

axes[3].imshow(predicted, cmap="gray")
axes[3].set_title("Frozen five-model ensemble mask")

for ax in axes:
    ax.axis("off")

fig.suptitle(
    f"{main_id} | same case, same axial slice | "
    "frozen local threshold 0.40"
)
fig.tight_layout()
fig.savefig(OUT / "03_frozen_member_vs_ensemble.png",
            dpi=300, bbox_inches="tight")
plt.close(fig)






fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, sid, scale_name in zip(
    axes[0], scale_ids, ("lower-volume", "middle-volume", "higher-volume")
):
    example = cases[sid]
    z = best_target_slice(example["target"])
    overlay(
        ax,
        image_slice(example["image"], z),
        image_slice(example["target"], z),
        image_slice(example["ensemble"], z),
    )
    volume = float(example["official"]["target_volume_ml"])
    ax.set_title(
        f"{scale_name}: {sid}\n"
        f"reference {volume:.2f} mL | "
        f"Dice {float(example['official']['dice']):.3f}"
    )

example = cases[component_id]
labels, n_components = ndimage.label(
    example["target"], structure=STRUCTURE_26
)
require(
    n_components == len(val_components[
        val_components["subject_id"] == component_id
    ]),
    "Reconstructed component count differs from official R3 audit."
)


visible_counts = [
    np.unique(labels[:, :, z][labels[:, :, z] > 0]).size
    for z in range(labels.shape[2])
]
z = int(np.argmax(visible_counts))

overlay(
    axes[1, 0],
    image_slice(example["image"], z),
    image_slice(example["target"], z),
    image_slice(example["ensemble"], z),
)
axes[1, 0].set_title(
    f"Component-rich case: {component_id}\n"
    f"{n_components} 3D target components"
)


subject_component_rows = (
    val_components[
        val_components["subject_id"] == component_id
    ].sort_values("component_id")
)

axes[1, 1].scatter(
    subject_component_rows["component_volume_ml"],
    subject_component_rows["local_maximum_iou"],
    c=[
        "tab:green" if detected else "tab:red"
        for detected in subject_component_rows[
            "local_detected_iou_ge_0p10"
        ]
    ],
    s=35,
)
axes[1, 1].axhline(0.10, ls="--", lw=1, color="black")
axes[1, 1].set_xscale("log")
axes[1, 1].set_ylim(-0.02, 1.02)
axes[1, 1].set_xlabel("Target-component volume (mL)")
axes[1, 1].set_ylabel("Maximum matched-prediction IoU")
axes[1, 1].set_title(f"{component_id}: official component audit")

tier_order = [
    "subvoxel_or_single_voxel", "micro", "tiny",
    "small", "medium", "large",
]
tier_counts = (
    val_components.groupby("component_tier")
    ["local_detected_iou_ge_0p10"]
    .agg(["sum", "count"])
    .reindex(tier_order)
    .fillna(0)
)
rates = [
    float(row["sum"] / row["count"]) if row["count"] else 0.0
    for _, row in tier_counts.iterrows()
]
axes[1, 2].barh(
    [name.replace("subvoxel_or_single_voxel", "single voxel")
     for name in tier_order],
    rates,
    color="steelblue",
)
axes[1, 2].set_xlim(0, 1)
axes[1, 2].set_xlabel("Fraction recovered (IoU ≥ 0.10)")
axes[1, 2].set_title("Validation component recovery by tier")

fig.legend(handles=legend, loc="lower center", ncol=2)
fig.suptitle(
    "Locked validation: lesion scale and component recovery"
)
fig.tight_layout(rect=(0, 0.05, 1, 0.95))
fig.savefig(OUT / "04_lesion_scale_component_recovery.png",
            dpi=300, bbox_inches="tight")
plt.close(fig)






provenance = []
for sid, selected in cases.items():
    provenance.append({
        "subject_id": sid,
        "split": "val",
        "input_npz_path": str(selected["npz"]),
        "input_npz_sha256": sha256_file(selected["npz"]),
        "frozen_prediction_archive": str(selected["archive"]),
        "frozen_prediction_sha256": archive_index[sid]["sha256"],
        "local_threshold": FROZEN_LOCAL_THRESHOLD,
        "ensemble_source": "frozen five-fold EMA packed local mask",
        "target_source": "manifest validation target_binary",
    })

pd.DataFrame(provenance).to_csv(
    OUT / "figure_provenance.csv", index=False
)

print("\nTRIAL8 VERIFIED VALIDATION FIGURES COMPLETE")
print("Validation cases visualized:", len(cases))
print("Test target arrays read: False")
print("Model constructed / inference / training: False / False / False")
print("Frozen local threshold:", FROZEN_LOCAL_THRESHOLD)
print("Output directory:", OUT)
print("Files:")
for path in sorted(OUT.iterdir()):
    print(" ", path.name)
